# Tiled sum along either axis in Triton

Reductions over long rows cannot load a whole row into one block. The standard pattern is a **2-D tile** of `BLOCK_M` rows by `BLOCK_N` columns that walks along the reduced axis, accumulating in `float32`, followed by one in-register reduction at the end. Written against strides, the same kernel sums rows of a matrix or rows of its transposed **view**, which is a column sum.

Write `tiled_sum(x, dim)`.

## Inputs
- `x`: contiguous CUDA tensor `(M, N)` of dtype `float16` or `float32`, $M, N \ge 1$; $N$ can be much larger than a block (up to $10^6$).
- `dim`: `1` to sum each row (output length $M$), `0` to sum each column (output length $N$).

## Output
- `float32` tensor of the sums, length $M$ for `dim=1` and $N$ for `dim=0`.

## Requirements
- One kernel that reduces along the second axis of a `(R, C)` view given by `(stride_r, stride_c)`. For `dim=0`, call it on `x.t()` (no copy: pass the transposed strides).
- Each program owns `BLOCK_M` rows and loops over the columns in steps of `BLOCK_N`, masking both the row and the column edges.
- Accumulate a `(BLOCK_M, BLOCK_N)` `float32` tile and reduce it with `tl.sum(acc, axis=1)` once, after the loop. `float16` accumulation is not allowed: it cannot even count past 2048 exactly.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from an equivalent PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def tiled_sum(x, dim):
    """x: (M, N) float16/float32 CUDA tensor. Return float32 sums along dim (0 or 1)."""
    raise NotImplementedError


@triton.jit
def _row_sum_kernel(x_ptr, out_ptr, R, C, stride_r, stride_c, BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], "dtype": "float32"}, 1], "e": {"__nd__": [6.0, 15.0], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], "dtype": "float32"}, 0], "e": {"__nd__": [5.0, 7.0, 9.0], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]], "dtype": "float16"}, 1], "e": {"__nd__": [5000.0, 5000.0, 5000.0], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], "dtype": "float16"}, 0], "e": {"__nd__": [5000.0, 5000.0, 5000.0], "dtype": "float32"}}, {"a": [{"__nd__": [[0.8999031186103821, -0.4583752751350403, -1.3140853643417358, 0.6586976647377014, 0.9554258584976196, 0.5246310234069824, -2.558269500732422, -1.5353412628173828, 0.6136307120323181, 1.494365930557251, -1.4945822954177856, 0.9703254699707031, 1.1190851926803589, 0.017410578206181526, -1.339133381843567, -0.6403135657310486, -0.36200037598609924, 0.6909022927284241, 0.8993356227874756, -0.986848771572113, -0.10213501006364822, -0.03492661938071251, -0.1748647689819336, 0.9459331631660461, -0.2716423571109772, -0.0700131207704544, -1.4837714433670044, 0.14615555107593536, -0.9387549757957458, 1.8856043815612793, 0.14862462878227234, 0.1838734894990921, 0.0986730232834816, -1.1710097789764404, -0.4353877604007721, 0.11089904606342316, 0.9117034077644348, 0.21966463327407837, -1.3220744132995605, -0.21426066756248474, -0.13816185295581818, -1.6847881078720093, 0.5395525693893433, 0.7637683749198914, -0.9124293923377991, 2.1853322982788086, 0.324837327003479, -0.5884791612625122, 1.9458446502685547, -0.5886527895927429, 0.14952750504016876, -1.812504529953003, -1.097822904586792, 1.750251054763794, -1.0016318559646606, -0.8418654203414917, -0.26036927103996277, -0.8247355222702026, -1.2520310878753662, -1.0220783948898315, -0.17003101110458374, 1.5131309032440186, 0.7696735262870789, 2.263970136642456, -0.8698101043701172, 0.17317888140678406, 0.25686517357826233, 0.5076766014099121, 0.7218186855316162, -1.7291629314422607, 1.4759763479232788, -0.013240644708275795, -0.4972066581249237, 0.25351643562316895, -0.5871797204017639, -1.2591904401779175, -0.37412479519844055, -1.9415563344955444, -1.127770185470581, 0.7236483693122864, 0.4756447672843933, -1.151097297668457, -2.2454750537872314, 0.17502057552337646, 0.36982715129852295, -0.05980798602104187, -1.0016217231750488, 0.05884239077568054, -0.36620163917541504, 0.1824212670326233, 0.5660898685455322, -0.04666045308113098, -0.2123861461877823, 0.5310394167900085, 0.26019051671028137, -0.8776059150695801, -0.10265259444713593, 0.6619710326194763, 0.566676676273346, -0.32494738698005676, -0.16475775837898254, -1.0546441078186035, 0.22098210453987122, 1.0684579610824585, 0.4071141481399536, -0.46759316325187683, 0.6242401003837585, -0.004906778689473867, 0.6311420798301697, -1.2471394538879395, -0.4167402982711792, 0.7652712464332581, 1.1058533191680908, -0.7801048755645752, -0.5019945502281189, 1.8491978645324707, 0.3650195896625519, -0.153121218085289, 0.48895737528800964, -0.5500261783599854, -1.6647868156433105, 0.026143142953515053, -0.34950289130210876, -0.6110222935676575, 0.007791154086589813, -0.8798443078994751, -0.4935612678527832, -1.8657894134521484, 1.1923428773880005, 1.4240317344665527, -1.1406865119934082, -1.5961689949035645, 0.3878454864025116, 0.7894242405891418, -0.4591422379016876, -1.1156907081604004, 1.8323572874069214, -0.6859242916107178, 0.36959096789360046, 0.2881930470466614, 0.3097412884235382, 1.2666064500808716, 1.3789180517196655, -1.4739190340042114, -0.12562070786952972, 1.0913655757904053, 0.17375719547271729, -0.6511379480361938, 0.7755193114280701, -0.5232162475585938, -0.18898096680641174, 0.8250610828399658, 0.5654955506324768, 1.880777359008789, 0.6875735521316528, 0.003970490302890539, 1.0332063436508179, -0.04454539716243744, 0.06852097064256668, 0.13934198021888733, -0.41248008608818054, 0.26471850275993347, 0.1666335165500641, 0.17127445340156555, -0.9556545615196228, 0.5732484459877014, 0.10141777247190475, -0.15393570065498352, -0.1486535668373108, -0.6692723631858826, -0.6993617415428162, 1.8623509407043457, 0.17674580216407776, 0.7380908131599426, 0.6799888610839844, -0.7537494897842407, -0.14892086386680603, -0.5416708588600159, 0.517824649810791, 0.35509219765663147, -0.9670684933662415, 0.9322242140769958, 0.9030817151069641, -1.3903950452804565, 0.3607134521007538, 0.2591469883918762, -1.7361066341400146, -1.160036325454712, 0.5330931544303894, -1.2110098600387573, -1.9539152383804321, 0.03853512182831764, -1.9655067920684814, -0.20679813623428345, 0.09442973881959915, -0.10538601130247116, 0.5674481987953186, -0.0816906988620758, -0.2834911048412323, -1.1433733701705933, 1.1357825994491577, -1.1710913181304932, 0.3445030450820923, -0.09653632342815399, -0.4260581433773041, -1.315321683883667, -1.6966569423675537, -0.09251436591148376, 0.6657434105873108, 0.05454370751976967, -1.1645617485046387, 0.9531631469726562, -2.605194568634033, -0.13056007027626038, 0.9154669642448425, 1.0347535610198975, -0.14629866182804108, 1.2095682621002197, 1.728361964225769, -0.27340182662010193, 1.104597806930542, 0.06376875191926956, 1.0828369855880737, -1.1844961643218994, 0.7466009259223938, 1.2563724517822266, 1.1375545263290405, 0.09971880912780762, -1.9823657274246216, 0.2861904799938202, -2.318258762359619, 0.14128269255161285, 0.20135077834129333, 0.6472941637039185, -0.5949511528015137, 0.6877020597457886, -0.3357521891593933, -1.1458567380905151, 0.17348985373973846, 2.262974739074707, 0.35219985246658325, 0.47316688299179077, 1.6408905982971191, 1.1046124696731567, 1.102184772491455, -1.4408499002456665, -0.7911708354949951, 2.235787868499756, 0.2224586308002472, -0.8280426263809204, 1.261252999305725, 0.3924678862094879, -1.9293192625045776, -0.6627941131591797, -1.1360650062561035, -1.2008289098739624, -0.7337416410446167, -1.3539620637893677, -0.45091667771339417, 0.23689255118370056, -0.20710228383541107, -0.2102249711751938, 0.9449697732925415, -0.2974686622619629, 0.1296825259923935, -0.8770400285720825, -1.1862388849258423, 0.51100093126297, -0.972798228263855, 0.13335035741329193, 0.8983229994773865, -0.4918181598186493, 2.117896318435669, 1.154932975769043, -0.774941623210907, 2.13230562210083, 0.5126007199287415, 1.0842838287353516, -0.4711441099643707, 0.3590772747993469, -0.23570898175239563, 0.3278779685497284, -0.6708052158355713, -0.4871984124183655, -0.7413039207458496, -2.210026502609253, -0.8632765412330627, 0.270134299993515, -1.2500736713409424, -1.0433697700500488, 1.3556078672409058, -1.7961822748184204, -0.24617260694503784, 0.4725686311721802, 0.6061264872550964, 0.3723907768726349, 0.2877083122730255, -0.07896576821804047, -1.1026699542999268, -2.212771415710449, 1.6808733940124512, -0.41940876841545105, 0.10577099025249481, -1.0259913206100464, 0.02224990352988243, -1.428452730178833, -0.8158507943153381, 0.19532310962677002, 1.770680546760559, 0.49967917799949646, 2.0179903507232666, 0.27029886841773987, 0.20965175330638885, 1.4535316228866577, 0.012672136537730694, -1.4806346893310547, 0.5149244070053101, 0.3988657593727112, 0.18802908062934875, 0.45602163672447205, -0.340660959482193, -0.08864977955818176, -1.288060188293457, 0.7480973601341248, -0.684158980846405, 1.2231075763702393, 0.10027065128087997, -0.8516147136688232, 0.6974550485610962, 0.3599795401096344, 0.43590420484542847, 0.05157991871237755, -0.5600208044052124, 1.4960922002792358, -1.552392840385437, -1.041038990020752, 1.2759385108947754, 1.7062774896621704, -1.9293212890625, 1.7610541582107544, 0.6014499068260193, -2.00620174407959, -0.6632086634635925, -1.1056885719299316, 0.07043773680925369, -0.2917822599411011, 1.2332061529159546, 0.18745777010917664, 0.8555373549461365, 0.997447669506073, -1.104976773262024, -0.14799921214580536, 0.47859784960746765, 0.2995857894420624, -0.657784104347229, 0.4879797101020813, -1.1764971017837524, -0.6216272711753845, 0.5223340392112732, 0.6654309630393982, -1.0732537508010864, 0.2159944325685501, 0.5570297241210938, 0.9360799789428711, 0.6457073092460632, 0.16403627395629883, -1.0509146451950073, -1.6835135221481323, -0.8572655320167542, -0.6963451504707336, -0.4555371403694153, 0.25345197319984436, 0.5250223278999329, 1.9980356693267822, -1.6953502893447876, -0.9134610295295715, -1.7069451808929443, -1.1620583534240723, -1.446288824081421, -0.9352262616157532, -0.9528447389602661, -1.2973151206970215, -0.749412477016449, 0.6836749911308289, 0.6057178974151611, -0.15176734328269958, 0.7778471112251282, -2.188178300857544, -1.0612519979476929, 0.7438154816627502, -0.44453996419906616, -0.6817123889923096, -0.45675763487815857, -0.4625297784805298, 0.704009473323822, 1.447244644165039, -1.3964546918869019, -1.1432619094848633, -0.6047202944755554, 0.23877355456352234, 0.6882261633872986, -0.08775756508111954, 0.3833174705505371, -0.30538854002952576, 0.37029939889907837, -2.07387638092041, 0.7120949625968933, 0.045047517865896225, -1.1478885412216187, -0.06779567152261734, -1.2625044584274292, 1.1089059114456177, 1.073300838470459, 0.0848100483417511, -1.7510160207748413, -2.063983678817749, 0.0727948397397995, -1.1467972993850708, 2.0426647663116455, -1.507251262664795, -1.5780147314071655, 0.9184601306915283, 2.0231704711914062, -1.669063687324524, 0.803791880607605, 0.38384389877319336, -0.4328260123729706, -1.3933830261230469, -0.7584786415100098, -0.4709814190864563, 0.5679681301116943, 0.5609595775604248, -0.37571558356285095, -2.0546727180480957, -0.6836041212081909, -0.5735498070716858, -1.2105238437652588, 0.29197072982788086, 1.272192120552063, 0.665062665939331, -0.41499340534210205, -1.666552186012268, 0.4583747088909149, 1.1336168050765991, 0.20790615677833557, 0.163664311170578, 0.3044619560241699, -0.13469940423965454, -1.089009165763855, 1.8226481676101685, 1.1215150356292725, 1.3628349304199219, 0.009091629646718502, 0.7252418994903564, -2.0238122940063477, -0.1765352487564087, -0.1620151400566101, -0.8795292973518372, -0.922248363494873, -1.7394911050796509, 0.3160697817802429, 0.14133892953395844, 0.45134201645851135, -0.6857211589813232, 0.8102511763572693, 2.148977518081665, -0.5179665088653564, 0.6146741509437561, -2.6837375164031982, 1.754376769065857, 1.4886592626571655, 0.06065455451607704, 0.7225856184959412, -1.5005122423171997, 1.131286859512329, -1.0014513731002808, -0.3416120111942291, -0.5470161437988281, 0.9327576160430908, 0.22310537099838257, 0.24521410465240479, 1.0738639831542969, -1.2151941061019897, 0.4829792380332947, 0.9331173896789551, 2.1121885776519775, 1.2370816469192505, -1.0399430990219116, 0.9409905672073364, -1.5432658195495605, -0.42526742815971375, -0.36796969175338745, 0.6471886038780212, -0.48888903856277466, -0.1507345587015152, 0.34696587920188904, 0.015107192099094391, 0.31262239813804626, -0.5860698819160461, 0.03495031222701073, 1.0742604732513428, 0.7831284403800964, 0.41479969024658203, 0.3535030484199524, -1.6817735433578491, 1.1966789960861206, 1.6724965572357178, 1.3022451400756836, -1.2942326068878174, -0.6232863664627075, -0.6558471322059631, 0.23779083788394928, -0.5155795812606812, 0.18819868564605713, 0.6934308409690857, 0.06432077288627625, 1.6394416093826294, 1.1536107063293457, 0.26563167572021484, -0.6244698762893677, 0.08083276450634003, -0.5325934290885925, -1.2438243627548218, 0.15908955037593842, 1.307270884513855, -0.40861353278160095, -0.9304059147834778, 0.08489415794610977, -0.6191263198852539, 0.9043327569961548, -0.24656984210014343, 0.07291946560144424, -0.187511146068573, 0.09801522642374039, -0.8479596972465515, 0.14680016040802002, -0.6895408630371094, -0.7100067138671875, 0.29836875200271606, -1.6988108158111572, -0.7515841722488403, -0.2632159888744354, -0.23131847381591797, 0.47007840871810913, 1.118277668952942, 0.05540956184267998, 0.024942995980381966, -0.12039241939783096, 0.04569206386804581, -0.08779995143413544, 2.937380313873291, -0.7223324775695801, 0.9342005848884583, 0.9290662407875061, 0.6426178812980652, -0.10218202322721481, 0.5072893500328064, 1.085374116897583, -0.8222247958183289, 1.181255578994751, 0.27646568417549133, 1.5324430465698242, 0.19275467097759247, 0.225307896733284, 0.2398947924375534, 1.2878522872924805, -0.29720813035964966, 1.4777028560638428, 0.2831805348396301, -1.604684829711914, -0.2553133964538574, 1.4359402656555176, -0.03268507122993469, -1.3366377353668213, 0.47181636095046997, -0.1753959208726883, 0.6676793098449707, -0.4114547371864319, 1.2054307460784912, 0.5911117196083069, 0.12867727875709534, -1.2888044118881226, -0.3759533762931824, 1.6365784406661987, 2.461146593093872, 0.8043172955513, -0.04201611131429672, 0.8933999538421631, -0.18157339096069336, -1.2520462274551392, -0.2233319878578186, -1.5462781190872192, -0.11772065609693527, 0.21148231625556946, 0.8776394724845886, 2.142606735229492, -0.23554906249046326, 1.0518031120300293, -0.08785352855920792, 0.8981180191040039, 0.17291632294654846, 0.061453528702259064, -0.12160354107618332, -0.8075142502784729, -0.8923971652984619, 2.1207849979400635, -0.4519121050834656, -0.1141229122877121, -0.699027955532074, -1.239908218383789, 0.6202710866928101, -0.8065798282623291, -1.3927572965621948, -1.1554834842681885, -0.1550513058900833, 0.8607157468795776, -0.5249208211898804, 2.4977715015411377, -1.2541661262512207, 1.0243734121322632, -1.7396317720413208, -0.16639791429042816, 0.15475785732269287, -1.0757285356521606, 0.035045016556978226, 0.11347280442714691, -0.22673399746418, 1.1841027736663818, -0.18521659076213837, 1.355492115020752, -0.9944339990615845, 0.23443466424942017, -0.09827530384063721, -0.6036983132362366, 0.30331557989120483, -0.6957701444625854, 0.5019524097442627, -1.223759651184082, 0.004288239870220423, -0.4606814682483673, 1.5303953886032104, -0.01700490713119507, -1.1904680728912354, -2.1827242374420166, -0.5535383224487305, -0.3185899257659912, 1.7100979089736938, 2.3178536891937256, 0.013037207536399364, -0.3308594524860382, 1.3968191146850586, -1.1285423040390015, -0.4913652241230011, 1.455488920211792, -0.04435897618532181, 0.03992754966020584, 0.10061267018318176, -0.8511046767234802, -0.6384986042976379, -0.06595558673143387, -1.65879225730896, -0.3902221918106079, 0.703824520111084, -0.19587136805057526, -2.3835530281066895, 0.12895779311656952, 0.7925776243209839, -0.2523709535598755, 1.3760493993759155, 0.3249971270561218, 0.14536908268928528, -0.38748422265052795, -1.5404058694839478, -1.0179729461669922, -0.575279176235199, -1.827839732170105, 0.600877583026886, -1.0819159746170044, -1.6162034273147583, 0.0171206071972847, -0.780884325504303, -0.8399583697319031, -0.5029340982437134, 1.6197550296783447, -0.6615617871284485, 0.4144236445426941, 0.2150653451681137, -0.13213010132312775, -0.21591006219387054, -0.5756670236587524, 0.8215908408164978, 0.47491052746772766, -1.1652508974075317, 1.6954865455627441, 1.3738999366760254, 1.534897804260254, 1.7814605236053467, -0.7402556538581848, 0.13519427180290222, 0.4030706584453583, -0.7339342832565308, 0.2864648103713989, -0.5419684648513794, 1.4307308197021484, -0.13700881600379944, -1.7071185111999512, -1.4293266534805298, -0.8566039800643921, -0.7225134968757629, 1.216440200805664, -0.8717086315155029, -1.494749903678894, 0.4862147867679596, 0.6418257355690002, 1.2611799240112305, -0.9460441470146179, -0.3007318675518036, -0.5822508931159973, 0.36769694089889526, 1.2012457847595215, -0.45043936371803284, -1.3434103727340698, 0.08650703728199005, -1.3707393407821655, -2.541722536087036, 0.28454267978668213, 0.9492698311805725, -1.7157375812530518, -0.006869330070912838, -1.038689136505127, -0.6974465250968933, 0.058501001447439194, -0.37954795360565186, -1.1146408319473267, 1.9081367254257202, -0.15841223299503326, 1.0266318321228027, -0.6831233501434326, 0.8704742789268494, 0.5657032132148743, -0.8921152353286743, -1.1389124393463135, 0.5938212275505066, -0.031848639249801636, -1.4403527975082397, -1.1494170427322388, -1.4370230436325073, 0.8036302924156189, 0.5587061643600464, -0.7483652234077454, 1.8650603294372559, -0.2549581527709961, 0.7426502704620361, 1.556613564491272, -0.8623237609863281, -0.6465769410133362, -1.0332574844360352, -0.14799988269805908, -1.8309932947158813, 1.3821425437927246, 0.7643731236457825, -0.0799730122089386, -0.4333830177783966, 0.6291009187698364, -1.0556741952896118, -0.8429441452026367, -0.1958383321762085, 0.5132637619972229, -1.3493880033493042, -2.1075692176818848, -1.2414186000823975, 0.3184157907962799, 0.6318437457084656, 1.1161473989486694, 1.088407039642334, -2.3566079139709473, 1.9803273677825928, -1.195037603378296, 1.3977832794189453, 0.1043664887547493, -0.3818780481815338, 0.5468370914459229, -0.7387341260910034, -1.9393256902694702, -0.2706186771392822, -1.1248379945755005, 0.534800112247467, -0.03687746077775955, -0.23443321883678436, -0.8537653088569641, -1.1263427734375, 1.00685453414917, -0.1887790411710739, 0.7323111295700073, 0.6601270437240601, -0.7505186796188354, -0.27323904633522034, 1.2058132886886597, -0.8728693127632141, -0.08649203181266785, 0.8504368662834167, 1.2942299842834473, -0.7061967849731445, -1.2057045698165894, -0.03805288299918175, -0.8950231075286865, 0.7048787474632263, 0.6489371061325073, -1.7708303928375244, -0.24275313317775726, -0.8262097835540771, -0.033612336963415146, 1.1155413389205933, 0.43447861075401306, 0.8978976011276245, 0.0027777496725320816, -2.6910126209259033, -2.9569592475891113, -0.31769460439682007, 0.744182825088501, -0.3085881173610687, 0.17100951075553894, -1.5078818798065186, 0.8876871466636658, -2.4846179485321045, -1.3491706848144531, -1.546278476715088, 1.0564754009246826, -1.4004573822021484, 1.209554672241211, -0.1911104917526245, -1.5221372842788696, -1.0479254722595215, 0.5509051084518433, -0.9176278710365295, 1.0885967016220093, -0.7233878970146179, 1.569963812828064, -0.751186728477478, 0.9699144959449768, 0.04679206758737564, -0.10635776817798615, 0.2956945598125458, 0.9242297410964966, 0.2530613839626312, -0.6040695309638977, -0.002265071962028742, 0.2286248803138733, -0.5424455404281616, 0.8305144906044006, -0.96439129114151, -1.3908461332321167, -0.06986145675182343, -1.0823297500610352, 0.9213810563087463, 1.228393316268921, -1.919921875, 0.04241737723350525, -0.3029714822769165, 0.6972513794898987, -0.19374139606952667, 0.8257158994674683, -0.05092581734061241, 0.7098743915557861, -0.9867958426475525, 1.4491692781448364, -1.5946506261825562, -1.4568476676940918, 0.3270825743675232, 0.964439868927002, 0.2709067463874817, -2.6652159690856934, -1.3865383863449097, 0.24557794630527496, 0.06523028761148453, 0.12412471324205399, -0.2938699424266815, -0.2669796049594879, 0.13816572725772858, 1.2594645023345947, 0.6779926419258118, -0.5075498819351196, 0.6025964021682739, 0.2775305509567261, -0.9836982488632202, 0.8962817788124084, 0.11050381511449814, -0.34924063086509705, 0.37133851647377014, -2.9414453506469727, 1.601850152015686, -0.31375640630722046, -0.44220975041389465, -0.9816311597824097, 0.30627667903900146, 0.7132523059844971, 1.1218878030776978, -0.6482441425323486, 1.9496804475784302, -0.4865599572658539, 0.010171583853662014, 2.970968008041382, 0.7807389497756958, -1.9831628799438477, 0.1812298446893692, 0.6112141609191895, -0.4037332832813263, 0.2940548360347748, 1.3554388284683228, -0.6453508734703064, -0.49437010288238525, 0.7139834761619568, -0.38205093145370483, -0.22394582629203796, 0.44720879197120667, 1.068036675453186, 1.0667312145233154, 2.111229181289673, 0.8928000926971436, 0.4409791827201843, -2.1375954151153564, -0.001538573415018618, -0.65567547082901, -1.6248083114624023, 0.02314361184835434, 0.45823681354522705, -0.6269470453262329, 0.7082387208938599, -1.1235982179641724, -0.7026557922363281, 1.1051366329193115, 0.5595276951789856, 0.4839227795600891, -0.34300899505615234, 0.7244430184364319, -1.2352527379989624, -0.05574558302760124, 0.08569443225860596, -0.26980888843536377, 1.271958589553833, 1.0944594144821167, 0.18767233192920685, 0.004210630431771278, 0.2315317541360855, 0.853624165058136, 0.34284618496894836, 0.24331901967525482, 0.4315861761569977, 1.1104165315628052, 0.5473667979240417, -0.4876408576965332, 0.5577390193939209, -0.9523491263389587, 1.203368902206421, 2.287093162536621, 0.15321217477321625, 0.9807662963867188, 1.265201210975647, -0.5547460913658142, 0.854436457157135, 1.3333178758621216, -0.26146236062049866, 0.8939447402954102, 0.5393257141113281, -0.9474541544914246, 0.8511086702346802, 1.8847929239273071, 0.24231652915477753, 0.48922330141067505, -0.8519030213356018, -0.9525532126426697, -0.4582779109477997, 0.3089849054813385, 0.4178001880645752, 0.43185341358184814, 0.9222221970558167, 0.5821750164031982, -0.13951098918914795, 0.8311170339584351, -0.7039746046066284, 1.8731147050857544, -0.7294390797615051, 0.04164654389023781, -0.9582239389419556, 1.4264137744903564, 0.5558668971061707, -0.7786097526550293, -0.794025719165802, 0.9583896994590759, -1.3961421251296997, 0.2524065375328064, -0.9657363891601562, 0.37706106901168823, 1.1542261838912964, 0.02713397517800331, 0.5405804514884949, 0.3851805031299591, -0.8401992917060852, -0.4251210391521454, -0.7389296293258667], [-1.179717779159546, -2.665703773498535, 1.4356073141098022, -0.491498202085495, 0.46894386410713196, 0.4630524814128876, -0.6732829809188843, 0.44945430755615234, 0.6006954908370972, -0.36795246601104736, -0.6248364448547363, 0.41569629311561584, 0.13297678530216217, -0.5198304653167725, 0.898234486579895, -1.3164219856262207, 1.0312918424606323, 0.10628420859575272, -1.1473209857940674, 0.41652318835258484, -0.7588562369346619, 0.03826719522476196, -0.15688541531562805, -0.6990015506744385, 1.4016379117965698, -0.08112846314907074, -0.20107074081897736, 0.8807601928710938, 0.4938201904296875, -0.04028857499361038, 0.22751930356025696, -1.2422022819519043, 1.3364770412445068, -1.5608032941818237, 1.522441029548645, 0.5861697793006897, -0.4113505184650421, -0.715015709400177, -1.796675682067871, -1.5641100406646729, 1.4067883491516113, 0.7288343906402588, 0.7648867964744568, 0.5806168913841248, -1.5306833982467651, 0.6830989122390747, -0.8719853758811951, -1.8474117517471313, -0.663332462310791, -0.5502601861953735, 1.2292487621307373, -1.1766968965530396, -1.116884469985962, -0.22315442562103271, 0.9783003926277161, -2.2504019737243652, -0.026549089699983597, 0.9600598812103271, 1.7288378477096558, 0.4192456305027008, 0.7869210243225098, -0.2834038734436035, -0.2535204291343689, 0.006409502122551203, -0.9464609622955322, 0.11732052266597748, 0.23467959463596344, 0.9794695973396301, 1.7882970571517944, 0.33955255150794983, 0.4407752454280853, 0.38020560145378113, -1.8028666973114014, 0.478577196598053, 0.034354954957962036, -0.8846963047981262, 0.4098007082939148, -1.2382882833480835, 0.8156172633171082, -0.6123212575912476, -1.0886752605438232, -0.35756537318229675, -0.10970030725002289, -1.2354501485824585, -2.0281107425689697, 1.2981054782867432, -1.284261703491211, -0.659424364566803, -0.35577183961868286, -1.3108446598052979, -0.4365514814853668, 0.3914957344532013, 2.1902589797973633, 1.455220103263855, 0.9997928738594055, -1.157912015914917, 0.960088849067688, -0.4760870337486267, 1.9024279117584229, -0.8372438549995422, -1.1885597705841064, 0.2792993485927582, 1.240312099456787, -0.8726211190223694, -1.1538082361221313, 0.29970884323120117, -1.222969651222229, -0.34200412034988403, -1.1053526401519775, -0.7955606579780579, 0.33235424757003784, 0.5915045738220215, 0.462802916765213, -1.1798439025878906, 0.05416422709822655, 0.24310925602912903, -1.1842577457427979, -0.4464147686958313, 0.6097335815429688, 1.0225083827972412, 1.207926630973816, -0.9957178831100464, -0.8028660416603088, -0.08736041188240051, -0.5841209292411804, -1.1596986055374146, -0.19880369305610657, 0.039243679493665695, 1.016602635383606, 0.0021594504360109568, -1.1832267045974731, 1.138392448425293, -0.7801547646522522, 0.9778569340705872, -0.05609746649861336, -0.6341299414634705, 1.9264453649520874, -0.4241088330745697, -2.276127815246582, -0.9432900547981262, -0.6659168601036072, 0.8874530792236328, -0.33722952008247375, -0.6951429843902588, -0.7948446273803711, -1.3862978219985962, -0.1582811176776886, 0.3939249813556671, 0.4637182354927063, -0.09862518310546875, -1.7092335224151611, -0.5752314925193787, 0.33637985587120056, 0.2500740587711334, 1.3591388463974, 1.2648379802703857, 1.0045390129089355, 1.777077078819275, 0.8857879638671875, 0.24294352531433105, 1.485949993133545, -0.5768786668777466, 0.8962158560752869, -1.8457896709442139, -0.09838689118623734, 0.9826818108558655, 1.8390215635299683, -1.9730743169784546, -1.2194558382034302, -0.22949755191802979, -1.3033612966537476, -0.468979150056839, 0.03609301894903183, 1.4470247030258179, -0.892238974571228, 0.5959594249725342, -1.5505070686340332, 1.156083583831787, 0.45588386058807373, 0.5596024394035339, -0.34148871898651123, 1.210231065750122, 0.7127923965454102, -1.7253141403198242, 1.3332270383834839, 2.0510940551757812, 0.5522165298461914, -0.1679176539182663, 0.2850044369697571, 2.28482723236084, -0.2683456540107727, -1.4254059791564941, 1.4899109601974487, 0.6456810235977173, -0.0008382191881537437, 0.5424531102180481, -0.5704626441001892, 1.505360722541809, 0.4919479489326477, -0.5281388163566589, -1.7758952379226685, -1.4390945434570312, -1.0993258953094482, 0.5244581699371338, 0.12036565691232681, -0.5530344843864441, 0.21207955479621887, 0.985340416431427, 0.0844932273030281, 1.2224406003952026, -0.9929238557815552, -0.16473734378814697, 0.8481938242912292, -0.3639611303806305, -0.14604413509368896, -2.2999157905578613, 0.49820196628570557, 0.5129743218421936, -0.6230924725532532, -0.49698272347450256, 0.1296909749507904, 0.9923047423362732, -2.2392148971557617, -2.616438388824463, 0.6562344431877136, 1.5428861379623413, 0.062160421162843704, -0.40183451771736145, -0.1819300800561905, -0.01330346055328846, 0.11034388840198517, 1.6083965301513672, -1.5473889112472534, 0.7326810956001282, 0.2484724223613739, 0.5079293847084045, -2.182305097579956, 0.8615733981132507, -2.32364821434021, -1.1983821392059326, -0.8048321008682251, -0.4721190929412842, 0.09090957790613174, -0.9109734892845154, 0.6957644820213318, 0.40681755542755127, 0.7599347233772278, 0.3173638880252838, -0.3694393038749695, 1.2629344463348389, -0.25047004222869873, -0.330195814371109, 0.4560188353061676, 1.0703057050704956, 0.8246799111366272, -0.31268805265426636, 0.8544406294822693, 0.32552021741867065, -1.805362582206726, -0.005000210367143154, -0.2121913731098175, -0.5081697702407837, -0.4858408570289612, -0.6500990986824036, -0.5463030934333801, 0.00952125247567892, 1.1016736030578613, -0.7539150714874268, 2.0666449069976807, -1.006325364112854, 1.6236298084259033, -0.6658770442008972, -0.0837690532207489, -0.4862793982028961, 1.8097050189971924, -0.3744311034679413, 0.36108261346817017, -1.3231927156448364, -0.30176061391830444, -1.089979887008667, -0.05505608394742012, -2.0198400020599365, -1.8138787746429443, 1.4165327548980713, -0.2320563644170761, -0.9649589657783508, 1.0070873498916626, -0.7442213892936707, 0.710289478302002, -0.2076931893825531, -1.116620421409607, 0.03305329382419586, -0.9776239991188049, -1.3432561159133911, -2.6633312702178955, 2.1998233795166016, -1.7679386138916016, -1.0564184188842773, -2.084948778152466, -1.9199737310409546, 0.8787534236907959, 1.5946288108825684, -1.158696174621582, -0.12082263082265854, 0.4983322024345398, 0.27485665678977966, 0.45786553621292114, 1.1695462465286255, -1.6934912204742432, 0.5562804937362671, 1.2203669548034668, 0.9696736335754395, 0.059353966265916824, 0.48526284098625183, 0.48054903745651245, 1.360451579093933, -0.1990768164396286, -0.1440541297197342, -1.8188444375991821, -0.0011682405602186918, -0.4306592345237732, 1.2280662059783936, -0.7029686570167542, 0.44393065571784973, 0.25271859765052795, -0.3569774925708771, 0.9649531245231628, 0.04803653806447983, 0.5708579421043396, -0.1743210107088089, -0.3089892566204071, -1.7813185453414917, 0.34780484437942505, -0.19395321607589722, -1.9013041257858276, -0.3555911183357239, -0.545335590839386, 0.8513015508651733, -0.4159545302391052, 0.22912654280662537, -0.3120248019695282, -1.2175706624984741, 0.024887073785066605, -0.41345661878585815, -1.2927535772323608, -0.9435593485832214, -1.2080312967300415, -0.3977668583393097, 0.487494558095932, 2.5930190086364746, 0.1864939033985138, 0.2801334857940674, 0.562183141708374, -1.180060625076294, -1.2601020336151123, 0.8675949573516846, 0.5094172954559326, 0.938113272190094, 0.14628809690475464, -1.6217007637023926, 0.07194868475198746, -2.0524168014526367, 0.5233405232429504, 0.2769964933395386, -0.8159721493721008, -0.8666795492172241, -0.9094231724739075, -0.8983151912689209, 0.3509356677532196, 0.5026132464408875, -0.7427237033843994, 0.13776817917823792, 1.419980764389038, -0.4051627516746521, 0.258024662733078, -0.9395707249641418, -2.0066866874694824, -0.3979945182800293, -0.7039204835891724, 1.2575139999389648, -0.8063749074935913, -0.11140891164541245, -0.12228339910507202, -0.274550199508667, 2.056575059890747, 0.0007191899348981678, -1.0769264698028564, -1.0662426948547363, -0.7396999597549438, -1.038934588432312, 0.08246397227048874, -0.6776779294013977, -0.7065669298171997, -1.1987769603729248, 0.6441663503646851, -0.10321833193302155, -0.6512730121612549, 0.7108118534088135, 0.6076623797416687, 1.2074064016342163, -0.9303717017173767, 1.4292367696762085, 0.7266244888305664, 0.1793835312128067, 0.8136462569236755, 0.7081722617149353, -0.9408215880393982, -1.330755352973938, -1.789583444595337, -0.920840859413147, -0.9281929731369019, 0.6046157479286194, -0.49416065216064453, -0.8374435901641846, 0.5765594244003296, 0.37432289123535156, 0.3760901391506195, 1.4074956178665161, -0.3568952679634094, -0.2807236611843109, 1.914574384689331, 1.3255313634872437, -1.792946696281433, -0.09823252260684967, 0.32115840911865234, 2.112454414367676, -0.01061192061752081, 0.5304145216941833, -2.3519225120544434, 0.04478298872709274, -0.860512375831604, 1.9200011491775513, -1.3756797313690186, 0.008779442869126797, -2.1717987060546875, 0.7402613162994385, -0.9781771898269653, 0.5742835402488708, 0.8983065485954285, -0.9302676916122437, 1.3348839282989502, 1.5149152278900146, 0.594433069229126, -1.1432583332061768, 0.09890557825565338, -0.5026213526725769, -0.23091302812099457, 0.03197353705763817, -0.26873883605003357, 0.25820693373680115, -0.5280998945236206, 1.431689977645874, -2.526808261871338, -0.6436015963554382, 0.1617695391178131, -1.284970998764038, -0.412663072347641, -0.2437162697315216, -1.4966198205947876, -0.6292728185653687, -0.04020804166793823, -1.8384084701538086, -0.06322100013494492, 1.3774735927581787, -0.0024771252647042274, 1.507949948310852, 1.1397091150283813, -0.21432091295719147, 0.515099823474884, 1.0319112539291382, -0.6079912185668945, 2.072019100189209, 0.6979082822799683, 0.255918949842453, -0.021805427968502045, -0.19099211692810059, -0.45064279437065125, -0.1313873827457428, 0.41438692808151245, -0.634353518486023, -1.1304303407669067, 1.5309072732925415, 1.590489387512207, -0.10195150971412659, -0.21800914406776428, -1.0979174375534058, -0.023040322586894035, -2.2529568672180176, 0.8131577968597412, 0.9166340231895447, -1.255744218826294, 0.1477547436952591, -1.6343077421188354, -0.27313435077667236, -0.60235196352005, 0.1792914867401123, 1.3425698280334473, 1.0641388893127441, 0.1138862892985344, 0.5967855453491211, -1.7611325979232788, -0.44235482811927795, -0.14458729326725006, -0.1922607570886612, 0.21417218446731567, 0.5368334054946899, -0.021568486467003822, 0.4227941632270813, -0.5910213589668274, -1.8227453231811523, -0.5585258603096008, 0.5273871421813965, 0.8045322299003601, -1.885867714881897, -1.4262117147445679, -0.617684543132782, -0.2348073571920395, 1.9817605018615723, -0.45795929431915283, 1.5475995540618896, 1.3176418542861938, -0.8948923349380493, 0.8774781823158264, 0.6532900333404541, -0.860852837562561, -0.2434271275997162, -0.5453047752380371, -0.31026092171669006, -0.6048600077629089, -1.192702054977417, -0.20955362915992737, -0.6233299970626831, 0.29834872484207153, -0.6292323470115662, 0.7884424924850464, -0.5981305241584778, 0.20838885009288788, -0.3676600456237793, 1.6633070707321167, -0.018315935507416725, 0.31944897770881653, -0.6153111457824707, 1.487412929534912, -2.3068196773529053, -2.049015760421753, -1.137660264968872, -1.6164379119873047, -0.8252266645431519, -0.6303850412368774, -0.028483035042881966, -1.4813190698623657, 0.6765457391738892, 0.7064183950424194, 0.6991832852363586, 1.1151520013809204, -0.670414388179779, -3.7679977416992188, 0.5549207329750061, 1.4494202136993408, -0.27025195956230164, 0.739774227142334, 0.03226511552929878, 1.3589082956314087, -0.12282413244247437, 0.38222771883010864, 2.450981855392456, -2.2131237983703613, -0.2488706111907959, 1.4983537197113037, -1.6479604244232178, 0.09928475320339203, 0.8983843326568604, -1.7217707633972168, -0.3000392019748688, 0.912118673324585, 0.6295017600059509, 1.4861860275268555, -1.713024616241455, 0.6913756728172302, -0.4565344452857971, -0.6052491068840027, 1.447505235671997, -1.737714409828186, 0.3383203148841858, 1.1490925550460815, -1.3965381383895874, 0.7470375299453735, -0.4334375262260437, -1.8461668491363525, 0.980688750743866, -1.8626978397369385, 1.383813738822937, -0.711910605430603, -0.2723172605037689, 0.39477482438087463, 0.7907900214195251, 0.9467736482620239, -0.8245117664337158, -1.489797830581665, -0.34852996468544006, -0.6639066338539124, 2.2239181995391846, 0.21247488260269165, 0.2475304901599884, 0.6061961054801941, -1.4521735906600952, -0.5143530964851379, -1.4138424396514893, 1.9953510761260986, -0.6346685886383057, 0.2459847331047058, 0.7155243754386902, 0.3113636076450348, 0.3626692593097687, 0.7222578525543213, 0.76004958152771, -0.4648844599723816, 0.5418962240219116, 1.5209969282150269, -0.4043193459510803, 2.213106870651245, -1.053531289100647, -0.2759469151496887, 0.8472574949264526, 0.1256166249513626, 1.580712914466858, 1.057334303855896, -0.02894216775894165, 0.4695548713207245, -0.11845706403255463, -1.211267352104187, 1.9116370677947998, 0.05762818083167076, 0.5327285528182983, -2.755430221557617, -0.9666861295700073, -0.509304404258728, -0.4227880835533142, 0.14325475692749023, -0.44659146666526794, -1.4629899263381958, 0.46694308519363403, 0.6687488555908203, -0.3058018386363983, -0.2474314272403717, 0.20040200650691986, 0.878034770488739, -0.9595992565155029, 1.4995898008346558, -0.7835356593132019, -0.08539485931396484, 0.3021692931652069, 0.8105026483535767, -0.8883257508277893, 0.16308173537254333, -0.8294118642807007, 1.4996343851089478, -0.0625474750995636, 1.5083374977111816, 1.039313793182373, 0.3828728497028351, -0.23036427795886993, 0.3595101237297058, -0.34171175956726074, 0.352510541677475, 1.0944455862045288, -1.0603115558624268, 1.1162198781967163, 0.14490365982055664, 0.22394491732120514, 0.6653387546539307, -0.14154288172721863, -0.14901958405971527, 0.19632019102573395, -1.8851860761642456, -0.835516631603241, -0.20279796421527863, 0.9377351999282837, -0.8374294638633728, 0.6385728120803833, 0.12492170184850693, 0.4615686535835266, 0.04279348999261856, 0.30709534883499146, 0.2576027512550354, -0.8271039724349976, 3.2502903938293457, -0.19153210520744324, 0.21993862092494965, 0.18383772671222687, 0.16326306760311127, -1.5419727563858032, -2.391817808151245, -1.1703927516937256, -0.36263731122016907, 1.0515587329864502, 1.8410060405731201, 1.0539880990982056, 2.069675922393799, 1.0463762283325195, -0.6795795559883118, -0.7563943266868591, 0.34195154905319214, -0.46421507000923157, -0.21407008171081543, -1.2966785430908203, 0.5747382044792175, 1.3843786716461182, -2.5175297260284424, 0.5084304213523865, -0.2783489227294922, -1.240092396736145, 0.1603882610797882, -0.8913378119468689, -0.08792746812105179, 0.02707221917808056, 0.7074143886566162, 0.9234904050827026, 1.2984203100204468, -0.22021719813346863, -0.45979249477386475, -1.5067737102508545, -0.505081057548523, -0.5878749489784241, 1.0580514669418335, 0.08533702045679092, 1.4125248193740845, 0.29294946789741516, -0.5791202187538147, 1.099886178970337, -0.035009145736694336, -0.6068607568740845, 0.24506793916225433, -0.2784549593925476, 0.12435968965291977, 0.9012701511383057, 1.127774715423584, -0.07393535226583481, -1.0988141298294067, -1.4582359790802002, 1.1060227155685425, -0.44124269485473633, -0.5298418402671814, -0.9505932927131653, -0.639820396900177, -0.7663716673851013, 1.2306078672409058, 1.4097703695297241, 0.7234137654304504, -0.719792366027832, -0.5610424876213074, -0.35913681983947754, -0.9751121401786804, 0.5336808562278748, -1.0024164915084839, -0.16312481462955475, 0.0009374450310133398, 0.031466878950595856, -1.7540545463562012, 0.6035351157188416, -2.1358070373535156, -0.12762746214866638, 0.11102542281150818, 0.15819939970970154, 0.5185714960098267, 1.39644193649292, -1.8108388185501099, -0.990541398525238, -2.3956949710845947, -0.8946470618247986, 0.647798478603363, -0.41940346360206604, 0.3882126212120056, 0.35430118441581726, 2.3599650859832764, -0.23136983811855316, -0.4936714768409729, -0.36463090777397156, 1.022636890411377, -1.0653650760650635, -0.2703433930873871, 0.9643120169639587, -0.2583121657371521, -0.526416003704071, 0.09918398410081863, -0.35272854566574097, 0.3935942053794861, -0.6569238305091858, 1.5015417337417603, -1.5121046304702759, -2.145592451095581, -1.065731167793274, 0.7704298496246338, -0.37895503640174866, 1.637758731842041, -1.077773928642273, 0.43484577536582947, -0.05734919384121895, 1.444725751876831, 0.0979139655828476, -0.06382862478494644, -0.3386208117008209, 0.9276825785636902, -0.37600061297416687, 1.1527037620544434, 0.08764442056417465, -0.7881914973258972, 0.3407512903213501, -0.2344415783882141, -0.4014061987400055, 1.1215636730194092, 0.33226141333580017, -0.8036428689956665, 1.5453014373779297, -0.4230853319168091, -1.2981587648391724, 0.4916646182537079, 0.02014278434216976, -1.212384581565857, -1.024172306060791, -1.1282484531402588, 1.3797791004180908, 0.9831665754318237, -0.7361054420471191, 0.0031751771457493305, 0.6935985088348389, 0.6855688095092773, 1.1869772672653198, 2.457927942276001, -0.897288978099823, -0.2037564069032669, -0.743502140045166, 1.3265676498413086, 0.11827308684587479, 1.0294450521469116, -0.1681685447692871, 0.037038106471300125, 1.4155890941619873, -0.024147426709532738, 0.12740236520767212, -1.256752848625183, 1.732814908027649, -0.658704400062561, 0.33684584498405457, -0.21729344129562378, -0.16970239579677582, -1.0838935375213623, -0.7002350687980652, -1.8287628889083862, 0.43022847175598145, 1.3552838563919067, -1.6214100122451782, -0.4875483512878418, 0.4143708050251007, 0.5146179795265198, -0.3156066834926605, -1.1812398433685303, -1.8035709857940674, -1.6437913179397583, 2.114628553390503, -0.6022742390632629, -2.1312084197998047, -0.6494404077529907, 0.6762714385986328, 0.3991154730319977, -1.4838532209396362, 0.1026792898774147, -1.2792972326278687, -1.2131582498550415, 0.6953432559967041, 1.0922316312789917, -2.2884328365325928, 0.6799390912055969, 0.18415388464927673, 0.40973392128944397, -0.12326642125844955, 0.3602060079574585, -0.18213045597076416, 0.3457046449184418, -0.3708389401435852, 0.8817909955978394, -1.5168603658676147, 0.7180557250976562, -0.8634173274040222, -1.9698121547698975, -1.1364762783050537, -0.6883548498153687, 0.9740656018257141, 1.4669350385665894, 0.17532455921173096, -1.044979214668274, 0.7143205404281616, 2.689962387084961, -2.578665018081665, 1.376423954963684, 0.37085410952568054, -1.2945469617843628, -0.20865505933761597, -0.8598510026931763, -0.11789493262767792, -0.24957609176635742, 0.28290244936943054, -0.784021258354187, -1.3176707029342651, 0.09295748174190521, 0.030073143541812897, 0.5392067432403564, 1.8996367454528809, -0.545200526714325, 0.9804509878158569, -0.39754199981689453, -1.0288605690002441, 1.2147682905197144, -1.8796896934509277, 0.8849246501922607, -0.7431023120880127, 0.02508961781859398, 0.021814610809087753, 0.12624585628509521, -1.771342396736145, -1.3708492517471313, -1.378656029701233, -2.095831871032715, -0.0691559687256813, -0.41842585802078247, -0.3496246635913849, 0.08096820116043091, -0.14532925188541412, 1.437666654586792, 0.5193088054656982, -1.4708075523376465, 0.4889136254787445, -1.2152122259140015, 0.9030336141586304, -1.2275418043136597, -0.8061491847038269, -0.3852047622203827, 0.4848991632461548, -0.41770437359809875, 0.4540065824985504, 2.6646270751953125, -0.6805050373077393, 1.0918060541152954, -1.6112881898880005, -0.2455417662858963, -0.6864802837371826, -1.2117727994918823, 1.2038301229476929, -0.2591373324394226, -0.28814172744750977, -1.6071385145187378, -1.43028724193573, -1.4036364555358887, -0.5248169898986816, 0.34404894709587097, -0.3734819293022156, -0.2872104346752167, 0.43879565596580505, -0.8108862638473511, 0.005336306523531675, -0.3300354480743408, -0.12913525104522705, 0.7253822684288025, 0.7893970012664795, -0.237039715051651, 1.173269271850586, 1.6224768161773682, -0.9495989680290222, 0.5134251117706299, 0.5040676593780518, -1.1526801586151123, 0.4949263334274292, 0.49130168557167053, -2.070441246032715, 0.8422613739967346, -1.4758999347686768, -0.8901575207710266, -1.911316990852356, -0.33437785506248474, -0.47482815384864807, 0.8722385764122009, -0.4562698304653168, -1.37643563747406, 1.4179246425628662, -0.17544561624526978, -0.3007773458957672, 0.9156194925308228, -0.9888858795166016, -1.8128583431243896, 0.2562413513660431, -0.2672420144081116, -1.227473497390747, -0.402447372674942, 0.6951013803482056, 1.3002431392669678, -2.2080814838409424, 0.10351710021495819, 0.20042270421981812, -1.2821789979934692, -0.729926586151123], [0.40107157826423645, 0.11248844861984253, 1.5171936750411987, -0.05625048279762268, 1.162052869796753, -1.5830533504486084, 0.13167443871498108, -0.47474050521850586, -1.6676690578460693, 0.7157323360443115, 1.5018305778503418, 0.26352810859680176, -1.7888318300247192, 0.08597050607204437, -0.31659382581710815, -1.4124858379364014, 0.8543019890785217, 1.2984063625335693, 0.13837644457817078, 0.8985907435417175, 1.06361985206604, 0.7507938742637634, 1.4183677434921265, 1.023157000541687, 0.6626275181770325, 1.6546343564987183, -0.8233789205551147, 2.005817413330078, -0.6204504370689392, 0.06315172463655472, -0.23119528591632843, 0.6624128818511963, 1.2157474756240845, 0.7406305074691772, -0.21093013882637024, -1.0813395977020264, 2.304866075515747, -1.3469252586364746, -0.08770667761564255, 0.7094663381576538, -0.5715884566307068, 1.368421196937561, -1.3708343505859375, 0.525062620639801, -2.228431463241577, 1.952713966369629, 2.638108253479004, -0.9800573587417603, -0.27161484956741333, 0.7141802906990051, -0.062568299472332, 1.5313307046890259, 0.8119798302650452, -1.1985061168670654, -0.8411993980407715, -0.731806218624115, -0.46005794405937195, 0.019985433667898178, -0.10329987853765488, 0.8273008465766907, 0.9678579568862915, 0.9882715940475464, -1.7561088800430298, -0.029074518010020256, 0.2534950077533722, -0.5133402347564697, 1.834403157234192, 1.673601746559143, 0.49277228116989136, -0.5848450660705566, -0.5493894815444946, 0.47072291374206543, -0.44195806980133057, 0.0907803401350975, 0.26928067207336426, 0.4881858825683594, 2.25488018989563, -1.3602641820907593, -1.5005815029144287, -0.09589898586273193, -0.5117193460464478, -1.2149391174316406, 0.24449960887432098, -0.23208093643188477, 0.1697874367237091, -0.6839438676834106, -0.663449227809906, -2.0876529216766357, -0.697304904460907, -0.6779983043670654, 0.030023908242583275, -0.8261248469352722, 0.9288634657859802, -0.021958010271191597, -0.2878284156322479, 1.7334332466125488, 1.1394572257995605, 0.4025708734989166, -0.8080047965049744, 0.6093148589134216, 2.4937400817871094, 0.7076113820075989, -0.393223375082016, -0.9662273526191711, -1.1197113990783691, -1.2541718482971191, -0.29387161135673523, -2.061673641204834, -1.3096446990966797, 0.196504145860672, 0.8062432408332825, -1.014363169670105, 1.085580825805664, 1.144005298614502, 0.4837956130504608, -0.5743128657341003, -0.5039026141166687, 0.8465922474861145, -0.9432703852653503, -0.6749463677406311, 0.346348375082016, 1.0394617319107056, 0.5261312127113342, 1.860133171081543, 0.5382035374641418, 0.7095589637756348, -0.4374868869781494, 1.7808033227920532, -1.1994032859802246, 0.2578565776348114, 0.003710675286129117, 1.5860525369644165, -1.7977465391159058, 0.6933899521827698, 0.07299965620040894, 0.7408111691474915, -0.06671296805143356, -2.286343812942505, -0.6706362962722778, -0.4070431590080261, -0.18183347582817078, 1.3550323247909546, -0.14667661488056183, -0.1749134659767151, -0.5882591605186462, -0.3224141299724579, -0.980094850063324, 1.8164191246032715, 1.2524516582489014, -0.1938246488571167, -1.971239686012268, 0.605668306350708, -0.6217864155769348, -0.4065493941307068, 0.3026559054851532, -0.5929405689239502, 0.21755218505859375, -0.82145094871521, 0.1995057612657547, 0.9547842144966125, 0.22981497645378113, -0.6596660614013672, -0.5865942239761353, -1.6826391220092773, -0.4429577589035034, -1.0463193655014038, -0.03405516967177391, -1.2811641693115234, 0.644148051738739, 1.1679718494415283, 2.295997142791748, 0.49946463108062744, 1.1549549102783203, 0.6597863435745239, -0.4566473662853241, -0.27927330136299133, 0.6340168714523315, -0.5666196942329407, -0.4418620467185974, -0.8041290044784546, -0.8639131188392639, 0.9051325917243958, -1.2173289060592651, -1.7691930532455444, -1.3990808725357056, 1.7209707498550415, -0.7921659350395203, 2.2339189052581787, -0.6516728401184082, -0.7325198650360107, 2.583237648010254, 0.7035468816757202, 0.0368322879076004, 0.6401119232177734, -1.3064908981323242, -0.6730167269706726, 0.06121606379747391, -1.7836315631866455, 1.8410627841949463, -0.7834659218788147, 0.21497222781181335, -0.2512907087802887, 2.9512500762939453, 0.8258538842201233, 0.8299887776374817, -0.8573771715164185, -1.9652012586593628, 1.6658973693847656, -0.9953504800796509, -0.12151817977428436, -0.5166004300117493, 1.321007251739502, -1.113547682762146, -0.9945765137672424, -0.15878714621067047, -1.2645045518875122, 0.705176055431366, -0.6546556949615479, -2.190046787261963, -1.2402119636535645, 0.9216005206108093, -0.22152233123779297, -0.2380335032939911, -2.119309425354004, -0.05773083493113518, -0.02946457266807556, -1.432288646697998, 1.2479020357131958, -0.7588814496994019, 0.7944391965866089, 1.2127660512924194, 1.0010355710983276, 0.2142678052186966, 1.2218976020812988, -0.44252803921699524, -0.8713173270225525, -0.7309157848358154, 0.6863226890563965, 0.22883771359920502, -0.5854735374450684, -1.8622945547103882, 0.13047955930233002, -0.7632626295089722, 1.2823892831802368, -0.5077354311943054, 0.7208672761917114, 1.323121428489685, -2.210054397583008, -0.46826601028442383, 1.2202192544937134, -0.15905001759529114, 0.09857494384050369, -0.7462381720542908, 0.3020547926425934, -0.28252890706062317, -0.048897720873355865, 1.230759859085083, -1.9943397045135498, 0.8300361633300781, 3.7575411796569824, -0.46699193120002747, -0.8483969569206238, -1.5228526592254639, -0.9330482482910156, 0.9225545525550842, -0.4808100163936615, 0.10179721564054489, -2.2259693145751953, 0.05433781072497368, -0.26724621653556824, 0.09602277725934982, 0.20396873354911804, 0.20479841530323029, -1.0090302228927612, -0.013549759984016418, -0.9637280106544495, -0.08468464016914368, 1.1313281059265137, -0.6971340775489807, -1.0856798887252808, 1.4179487228393555, 1.6853034496307373, -0.3449119031429291, 0.7436985373497009, 1.4514081478118896, 0.09272009134292603, 1.586503267288208, -0.17085234820842743, 0.26527172327041626, -1.000540018081665, -0.9464293718338013, -0.14191733300685883, -1.1170077323913574, -0.6841689348220825, -0.8874569535255432, 1.2730120420455933, -1.107324481010437, 0.6400001645088196, -0.628736138343811, -0.2108902782201767, -0.49769070744514465, -1.1219767332077026, 2.0365638732910156, 0.4585055410861969, -0.8124166131019592, -1.4190102815628052, -1.0102483034133911, 0.07199376821517944, -0.34911245107650757, 0.03120686672627926, -1.2435638904571533, -0.5446205735206604, -0.21722254157066345, -0.934386134147644, -0.1185380145907402, -0.5104355216026306, 0.6091113686561584, -0.04846624284982681, 1.269591212272644, 1.8234503269195557, -0.12896761298179626, -0.996695339679718, -0.2065039873123169, 1.108787178993225, 0.6029279828071594, -0.5478240847587585, -0.5212636590003967, -0.23756884038448334, -0.9419698715209961, -1.0419690608978271, -1.16730535030365, -0.29786044359207153, -1.102216362953186, 0.43141257762908936, 0.9966529011726379, 0.7729189991950989, -0.6717371344566345, 1.8373103141784668, -0.37766000628471375, 1.1194566488265991, -1.077149510383606, -0.11799504607915878, 1.8300775289535522, 0.022798944264650345, -0.06551290303468704, 0.6136327385902405, -1.609268307685852, 1.218510389328003, -0.6036399006843567, -0.3058144748210907, 1.687220573425293, 0.6324081420898438, -0.7054689526557922, -0.6047933101654053, -0.3377457559108734, -0.37432658672332764, 0.31795868277549744, 1.0969336032867432, -0.917602002620697, -0.03657042235136032, 0.5233070850372314, 0.9182479381561279, -0.8860645294189453, -1.6849993467330933, 0.3372024595737457, 1.5494998693466187, 0.5091332793235779, -0.3024965524673462, -1.1873646974563599, 1.4508296251296997, -1.3742297887802124, 0.34418976306915283, 1.310017704963684, 0.20195774734020233, 0.1554163098335266, 1.0745983123779297, -1.5146942138671875, -0.6049176454544067, -0.7351235151290894, 1.084187388420105, 0.8891127705574036, -0.35399699211120605, 1.950961709022522, 0.9877452850341797, -0.13326673209667206, -0.2075616717338562, 0.9928106069564819, 0.831153392791748, 0.08828262239694595, 0.16622860729694366, 0.6299760341644287, 0.2295009344816208, 1.2608602046966553, 0.591141402721405, -0.43176138401031494, -1.450405478477478, -1.0189845561981201, -0.739018440246582, -0.30841341614723206, 0.9905819296836853, -0.6460029482841492, -0.14006198942661285, -0.21028642356395721, 1.0718525648117065, -0.19503161311149597, -0.5414747595787048, -0.37144431471824646, -0.3351280987262726, 0.5114478468894958, 0.474945992231369, 0.37564346194267273, 0.5163120627403259, 0.4124102294445038, -0.27548131346702576, 0.5331630110740662, 0.41746777296066284, -0.5865136384963989, -1.3342688083648682, -0.5657451152801514, -0.4520595967769623, -1.2742491960525513, -2.032310724258423, -0.16981783509254456, 0.6690875291824341, -0.008969045244157314, 0.10773415863513947, -1.0828536748886108, 0.0085885189473629, 0.701532781124115, -0.0630626529455185, -0.73561030626297, 0.7569798231124878, -0.6334829926490784, -0.15557312965393066, -1.120652675628662, 0.1579175442457199, 0.36152297258377075, 0.012591136619448662, -0.8080368041992188, -0.46157991886138916, 0.21639692783355713, -0.09482179582118988, -0.22171662747859955, 1.8458296060562134, 2.478273391723633, 0.1350417584180832, 0.9387320876121521, -0.10986661911010742, 0.8777616024017334, 1.5539265871047974, 0.34063130617141724, -0.35331013798713684, 0.20970883965492249, 0.37985625863075256, -0.3012240529060364, 0.20893658697605133, -0.7905332446098328, 0.7114876508712769, 0.7298285961151123, -0.2809576988220215, 0.27354779839515686, 1.4109400510787964, -0.5502831935882568, 0.779044508934021, -0.04133983701467514, 0.4400462508201599, -0.5696076154708862, 2.109607219696045, -0.6122036576271057, 1.0780926942825317, -0.7593865990638733, -1.1546369791030884, 0.01904722861945629, -0.20769155025482178, -0.6423287391662598, -0.7379605770111084, -0.5655580759048462, 0.984368622303009, 0.16269102692604065, 0.2727546691894531, 0.5281720161437988, 0.5240306854248047, 1.1009448766708374, 0.5318252444267273, -1.5854336023330688, -0.352181613445282, 0.7004758715629578, 1.3182944059371948, 0.05420364439487457, 0.18721118569374084, -0.22967472672462463, 0.7112058401107788, 1.1726067066192627, 0.04680094122886658, 0.3065390884876251, -0.7644605040550232, -0.9042505621910095, 0.42656898498535156, 1.1022566556930542, -0.3991999626159668, -0.2379390150308609, 0.08623664081096649, -0.360666424036026, -0.2719908356666565, 0.9519915580749512, 1.3257395029067993, 0.6603543162345886, -0.2601564824581146, 1.022500991821289, 0.7164359092712402, -1.0427571535110474, 0.48402711749076843, 0.8427267670631409, 0.21236921846866608, 2.0899717807769775, 0.9459696412086487, 0.9011909365653992, 0.3307592272758484, 0.9071002006530762, -0.9317687749862671, -1.2943469285964966, 0.7911126613616943, -1.8438796997070312, 0.9963704347610474, -0.644330620765686, 0.819264829158783, -1.5665993690490723, -0.5940564274787903, 1.2286591529846191, 1.493628740310669, 1.3109318017959595, 0.5686963200569153, -1.4720616340637207, 0.6178896427154541, -0.18978016078472137, -0.18092814087867737, -0.3438883423805237, 1.7818210124969482, 1.6354472637176514, -0.14776656031608582, 0.25841113924980164, -0.04556020721793175, -0.9596010446548462, -0.03289930522441864, 1.037282109260559, -0.8466551899909973, -0.47087839245796204, 1.8749223947525024, 0.4546196162700653, -1.516983151435852, 0.754109799861908, 1.6816319227218628, -0.14924365282058716, -0.6934044361114502, 0.39501163363456726, 1.0393571853637695, -0.21611429750919342, -0.2467990219593048, -2.6200971603393555, -1.9792057275772095, -1.309715747833252, 0.14208142459392548, 1.006695032119751, 0.70172518491745, 0.9722415804862976, -0.8592473268508911, -0.4835117757320404, -1.098766803741455, -0.7759919166564941, 1.0016374588012695, 0.16032874584197998, -0.2587509751319885, -0.5948218703269958, 0.5651354193687439, 1.0164674520492554, 1.1092535257339478, -0.4605601727962494, 1.7539442777633667, 0.6694806218147278, -0.676331102848053, -0.07376173138618469, 1.2833938598632812, 1.4656051397323608, -0.450529545545578, -1.0701922178268433, -0.6440037488937378, 0.34168320894241333, -0.6756145358085632, 0.29671505093574524, 1.5783675909042358, 0.8688259124755859, 0.4775187373161316, -0.6283763647079468, 1.4007364511489868, -1.2984215021133423, -0.0011815647594630718, 0.5408336520195007, 1.2541484832763672, -1.8811681270599365, -1.575905442237854, -0.5601462125778198, 2.255281686782837, -0.04707985743880272, 0.5321341753005981, 1.566293716430664, -1.4594324827194214, -0.47837722301483154, 0.10580122470855713, 0.6754146218299866, 0.2063751518726349, -0.665755569934845, -0.14942653477191925, 0.33964693546295166, -0.5366119742393494, 0.055525023490190506, -0.44056177139282227, -0.5111770033836365, -1.7284820079803467, 0.2210964858531952, -0.8771241307258606, -0.04530470445752144, -0.7823072671890259, -1.2890735864639282, -0.862220048904419, -0.6011978387832642, -0.020903071388602257, -0.10028854012489319, -0.5430030226707458, 0.01554049551486969, 0.10107069462537766, 0.08623089641332626, 1.115495204925537, -0.056531719863414764, 0.2520125210285187, 0.08887916803359985, -0.03696179389953613, -0.42896780371665955, -0.7431235313415527, -1.3119257688522339, 0.33971694111824036, -0.29489365220069885, -0.22762563824653625, 0.7245655655860901, 0.7632038593292236, 0.6924785375595093, 1.2822744846343994, 0.33860987424850464, -0.7426949143409729, 0.7297540903091431, 0.6514515280723572, 1.1964656114578247, -0.17519479990005493, 0.6640900373458862, -1.2036164999008179, 0.5920833945274353, 0.6605721712112427, 1.4935624599456787, 1.4716242551803589, 0.3876299262046814, 1.2174279689788818, 0.552365243434906, 0.2945398688316345, 1.5597779750823975, 0.4132893681526184, 0.7978946566581726, -0.16306520998477936, -0.3453797996044159, 0.6241661906242371, -0.8171179294586182, 1.9362022876739502, 1.5046889781951904, -1.3194129467010498, 1.2019692659378052, -1.2878085374832153, -0.14486850798130035, 0.6767037510871887, -0.31237009167671204, 0.9981086254119873, -0.19530817866325378, -1.8152176141738892, 0.9754126071929932, -0.1876583695411682, 1.437214970588684, -0.3460627794265747, -1.0855785608291626, -0.2759855091571808, 0.49442726373672485, -2.136962652206421, 1.8169363737106323, -0.3671073615550995, 0.4259355962276459, 0.21883925795555115, 1.5929914712905884, 0.15204942226409912, -2.1361606121063232, 0.7942264080047607, 0.2651371359825134, 0.14422249794006348, -1.2239391803741455, 0.8267810940742493, -0.9230107069015503, 1.123164415359497, 1.4918309450149536, 0.3459322452545166, 0.3403257131576538, 0.1072925478219986, 2.093668222427368, -0.47250956296920776, 1.3616018295288086, -1.2418612241744995, -0.8002734184265137, -1.8122563362121582, 0.03704371303319931, -1.5730395317077637, 0.574286937713623, 0.4278097450733185, -1.0498722791671753, -0.5534213781356812, -1.0351301431655884, -0.4824362099170685, 0.6159270405769348, -0.3318521976470947, -0.7905750870704651, 0.013681449927389622, -1.450927734375, -0.21188265085220337, -0.1774272620677948, -1.429725170135498, 0.21854358911514282, -0.12850825488567352, 0.948599636554718, -0.5846222639083862, -0.9605821371078491, 0.8891125321388245, 0.7549342513084412, 0.1392548829317093, -0.902070164680481, -0.7359170913696289, 0.5266828536987305, -0.6730678677558899, 1.1058584451675415, 0.020072191953659058, -1.0594167709350586, 1.3594410419464111, 1.607681155204773, 0.4355831444263458, -0.22123484313488007, -0.18188869953155518, -0.3594989478588104, 0.037390079349279404, 0.5025942325592041, 0.2918994724750519, -0.4856294095516205, 1.3691530227661133, -0.21753084659576416, 2.0714495182037354, -0.9945359826087952, 0.9617483615875244, 0.014530928805470467, 0.947394609451294, -0.5744219422340393, -1.0477172136306763, -1.7359687089920044, -0.5857177972793579, 0.13518157601356506, -0.8992565870285034, 0.44859060645103455, 1.0830693244934082, 0.8345542550086975, -0.06412634253501892, -1.2746319770812988, 0.6428699493408203, -0.5165280103683472, -0.7929400205612183, -0.2462347447872162, 0.3125182092189789, 0.3748677670955658, -0.11998848617076874, -0.9997764229774475, -1.0564124584197998, 0.5960147380828857, 0.26356688141822815, -0.41538000106811523, -1.13937246799469, -1.6378142833709717, 0.1507745087146759, 1.005835771560669, 0.1357908993959427, -1.0524604320526123, 1.5208725929260254, -1.9510256052017212, -0.7427549958229065, -0.7636708617210388, 0.19022785127162933, -0.7258796095848083, -2.1682193279266357, 0.9184992909431458, -0.9415436387062073, 0.2763286530971527, -0.780536413192749, 1.4110639095306396, -1.5974417924880981, 1.974958062171936, -1.0856789350509644, 2.157703399658203, 1.1058357954025269, 2.17228627204895, 1.866078495979309, -0.24190212786197662, -0.9003875851631165, 1.5390464067459106, 0.47608843445777893, -1.2149338722229004, -1.7279694080352783, -0.4448949098587036, 0.5574769973754883, -1.2485790252685547, -0.7256300449371338, 0.6946454644203186, -0.9892962574958801, -0.2594333589076996, 1.338673710823059, -1.4280219078063965, 1.1843323707580566, 0.31568512320518494, 1.02118980884552, -3.09879469871521, 1.5253710746765137, -0.7671810984611511, -0.9793148636817932, 0.2448863536119461, 1.2907251119613647, 0.06015590578317642, -1.6980342864990234, 1.4137508869171143, 0.15094105899333954, 1.046862244606018, 1.0392628908157349, -0.32319456338882446, -1.2166028022766113, 0.29989761114120483, 0.5807323455810547, 0.5006057024002075, -0.3911826014518738, 1.3174676895141602, 0.3667182922363281, 0.17916399240493774, 1.5232495069503784, 0.6737160682678223, 0.8948745131492615, -0.33569005131721497, -0.41660773754119873, -1.5355099439620972, 1.6363356113433838, -1.6590888500213623, 1.0900300741195679, -1.398013949394226, 0.7475332021713257, 0.7482397556304932, 0.7281690835952759, -0.6074103116989136, -0.2798879146575928, 0.9788053035736084, -0.5724311470985413, 0.8041585683822632, -1.0203309059143066, -0.9856090545654297, 1.569303035736084, 2.149243116378784, 1.5257282257080078, -0.3930453658103943, -0.5309750437736511, 0.9312905669212341, 0.5511982440948486, 0.11713720858097076, 0.0029402594082057476, 1.3937196731567383, 0.49271219968795776, -0.02808060124516487, -0.20819970965385437, -0.2503089904785156, -1.241460919380188, 1.2265700101852417, 0.007894786074757576, -1.6700396537780762, -0.7983174920082092, -0.4583938717842102, 0.05360901355743408, 1.570939064025879, -1.1148244142532349, -0.8898813724517822, -0.9318233132362366, 0.04167100787162781, -0.1704513281583786, -3.1053059101104736, -0.10652477294206619, -0.6032898426055908, -0.399217814207077, -0.7045793533325195, 0.6333852410316467, -1.212041974067688, 0.29239121079444885, -0.18179109692573547, 1.3543646335601807, -2.4340851306915283, -0.20719781517982483, 0.9852440357208252, 0.45120495557785034, -0.08513346314430237, -0.28626638650894165, -0.6989208459854126, 0.02041679620742798, -1.4590871334075928, -0.18408291041851044, -0.34965381026268005, 0.7216296195983887, -0.14063112437725067, -1.7219278812408447, 0.797718346118927, -0.3235187232494354, 0.8392775654792786, 0.9606361985206604, 1.2364064455032349, 1.118633508682251, -1.210412859916687, -0.6867432594299316, 0.33344703912734985, -0.0530235581099987, 0.5973663926124573, -1.2463345527648926, 0.2534125745296478, -0.7202317118644714, 1.14622962474823, -0.24326181411743164, 0.5368741154670715, -1.0494822263717651, 1.4175360202789307, 0.043993156403303146, -0.9838204383850098, 0.758858323097229, 1.5793057680130005, -0.11655687540769577, 1.9637420177459717, -0.6975983381271362, 0.1137014850974083, -0.4138297140598297, 0.43363022804260254, 0.7406242489814758, -1.0598641633987427, 0.229350283741951, -0.9108262658119202, 0.45690053701400757, -0.8669142723083496, -1.0755616426467896, 0.13132734596729279, -0.408875048160553, -1.1348172426223755, -1.1636073589324951, -1.4616849422454834, -1.4716285467147827, 0.7339748740196228, -0.6436705589294434, 0.07269831001758575, -2.344944715499878, -0.14216892421245575, 0.0062080384232103825, 0.05107325315475464, -0.24638426303863525, 0.44922059774398804, -2.09222412109375, -0.670089602470398, 1.245070219039917, -0.2361678183078766, -0.1035064160823822, 3.6640148162841797, 1.7282503843307495, 0.6571133732795715, -0.1402338296175003, -0.6737102270126343, 0.07283665984869003, -0.7320036888122559, 0.3420828878879547, -0.5979408621788025, 0.013463586568832397, -0.7292788028717041, -0.1334851235151291, -0.1413317620754242, -1.2083656787872314, -0.1230081245303154, -0.8350988030433655, -0.42145299911499023, -0.2895812690258026, -1.3793940544128418, 0.35791879892349243, 0.24459724128246307, -0.8562648892402649, -1.0982460975646973, 0.6396048069000244, 1.3979591131210327, -1.053088903427124], [0.10936755686998367, 1.4218318462371826, 0.9678647518157959, 2.4593923091888428, -0.4806312620639801, -0.6072132587432861, -1.1947648525238037, 0.06972446292638779, 1.2912312746047974, -1.0667694807052612, 0.10151305794715881, -0.3949592113494873, -0.8640904426574707, -0.6457409858703613, -0.5671964287757874, -1.7500213384628296, -0.5317729115486145, -0.41117021441459656, -0.6758841872215271, -0.4630023241043091, 0.9152907133102417, -0.9752108454704285, 0.13157658278942108, 0.9673910737037659, -1.231221079826355, 1.7662363052368164, -1.823956847190857, -0.3850262761116028, 0.5621654987335205, -0.16995283961296082, -0.26935964822769165, 1.415971040725708, 1.1519259214401245, -0.933907151222229, -0.5798577070236206, 1.9327162504196167, 0.6194162368774414, 0.38547512888908386, 1.1952109336853027, -0.3759394586086273, -0.4196530282497406, -0.07429429143667221, 0.9007154703140259, 0.6696605086326599, -0.7281674742698669, 1.30608332157135, -0.20198164880275726, 2.9270763397216797, -0.41142794489860535, 0.8790708184242249, -1.9834808111190796, -0.4311593174934387, -1.5148568153381348, -1.0679436922073364, 0.7250745892524719, -1.6210922002792358, 0.9958165287971497, 0.7441652417182922, -0.7882323265075684, 0.5373652577400208, -0.6139528155326843, 0.6588292121887207, 0.23751311004161835, 0.7906549572944641, -2.3296992778778076, -2.031126022338867, 0.589658260345459, -0.17000220715999603, -1.1054574251174927, 1.569218397140503, -1.9500060081481934, 2.072145462036133, 1.0490366220474243, -0.43739596009254456, 1.663769006729126, 0.4400269091129303, -0.4579814374446869, 0.21640408039093018, -1.8854526281356812, 0.4417535364627838, -0.6903690695762634, 0.47022807598114014, -0.3895149230957031, -0.4373016059398651, 0.2726907730102539, 1.0872281789779663, -0.01909545063972473, 0.607519268989563, -2.464831829071045, -1.7101333141326904, 1.657670021057129, -0.8716546893119812, -1.6895188093185425, -0.7895626425743103, -0.9290140271186829, 1.508525013923645, 1.4036284685134888, -0.23755072057247162, -0.5631338357925415, -0.5158030986785889, 1.3873507976531982, -1.198851227760315, -1.6062389612197876, -1.487395167350769, -0.3538319766521454, 1.117097020149231, -1.6440178155899048, 1.0789886713027954, -0.5433033108711243, -0.3179700970649719, 0.12425066530704498, -0.05559340864419937, -2.264462471008301, 2.429957389831543, -0.2926662266254425, -0.7138704061508179, -0.918107807636261, 0.22878092527389526, 0.11786015331745148, -0.5116540789604187, -0.8582608699798584, 1.7566680908203125, -0.5813036561012268, 0.4918835759162903, -0.2022688090801239, -0.8058778047561646, -0.63057941198349, 1.4461710453033447, -1.6421838998794556, -0.1111140251159668, -1.1836652755737305, 1.154997706413269, 0.9427019357681274, 0.07050982862710953, 0.578978955745697, -1.533183217048645, 1.0563734769821167, 1.025160312652588, 0.7964121103286743, -1.3578532934188843, 1.417523741722107, -0.16682256758213043, -2.263535737991333, 0.9802421927452087, 0.6105175018310547, -1.5634444952011108, 2.191577911376953, 0.7925490140914917, 0.8642694354057312, 0.22017601132392883, 1.0302220582962036, -2.156918525695801, -0.2931259274482727, -0.144461527466774, 0.41816744208335876, 1.680561900138855, -0.46954694390296936, 0.4291965663433075, -1.074721336364746, -0.4903208911418915, 0.8734135627746582, 0.1788935512304306, -1.1942930221557617, -0.9327466487884521, -0.25638964772224426, -1.0826427936553955, -0.38776034116744995, -1.165462851524353, -1.1282427310943604, -0.4040403664112091, -0.0850614458322525, 1.4715311527252197, -0.7339051961898804, 1.182477593421936, -2.1911840438842773, 0.12064320594072342, -1.871583342552185, -0.5847625136375427, 0.5013070106506348, 0.0446026585996151, -0.4075547456741333, 0.949201762676239, -1.700259804725647, -0.13947871327400208, 1.0736174583435059, -0.7368273735046387, -0.28714582324028015, 1.184221625328064, -0.7876341342926025, 1.2922176122665405, 0.058138009160757065, -0.423669695854187, -0.8614294528961182, 0.6368367075920105, 1.6361660957336426, -2.243959665298462, 2.631810426712036, -0.3393356502056122, -0.09503275156021118, -0.8179579973220825, -1.2120676040649414, 0.6494922637939453, -1.7295782566070557, 0.2986627221107483, -0.7859243750572205, -0.10322856158018112, -0.6283469200134277, 0.7088385820388794, 1.322974681854248, 0.29603174328804016, -1.0060392618179321, -0.08539615571498871, -1.3594313859939575, -2.4176902770996094, 0.12756936252117157, 0.38068047165870667, -0.91008460521698, 0.002672482980415225, -0.04392004758119583, -0.8426980972290039, -0.9130293726921082, -0.025842798873782158, 0.5816428065299988, 1.0986531972885132, 0.39145514369010925, 0.7380170226097107, -1.4038331508636475, -1.3520005941390991, 0.6645359992980957, -1.2096714973449707, -0.6510354280471802, 0.2867007851600647, 0.8583133816719055, 0.8945786952972412, -0.3836454749107361, 0.468523234128952, 2.0307819843292236, 0.37245655059814453, 1.060547113418579, -1.0211269855499268, -1.100699782371521, 1.4606682062149048, 0.5242165327072144, 0.8691370487213135, -1.6984246969223022, -2.638676881790161, 0.03080044686794281, -0.4149455726146698, -0.2504291832447052, -0.5746155381202698, 1.0549793243408203, 0.47744378447532654, 1.7973991632461548, -0.9096495509147644, -0.3227140009403229, -0.13451696932315826, -0.42088234424591064, 2.856851577758789, 1.0292963981628418, 0.5245506167411804, -0.5371926426887512, -0.9079551696777344, -2.0850069522857666, -1.6557585000991821, 1.48731529712677, -0.4345776438713074, 0.3613467514514923, 0.8643251061439514, -2.197402000427246, 0.3389036953449249, 1.1320533752441406, -1.2814130783081055, -0.6208911538124084, -1.5836894512176514, -1.9882844686508179, 0.0022190995514392853, -0.38683080673217773, 1.0104882717132568, 1.0024493932724, -0.16466820240020752, 1.130163311958313, 0.6545385718345642, 1.7192281484603882, 1.4566140174865723, -0.6041373014450073, 1.9719364643096924, 0.8945356607437134, -0.7342209219932556, -1.778987169265747, 0.3683503568172455, 0.2274673730134964, -1.1464083194732666, 0.31683114171028137, 2.335076093673706, 0.9457080960273743, -1.8271437883377075, -0.22689183056354523, -0.2992511987686157, -0.45797455310821533, -0.3293705880641937, -0.3317406475543976, 0.8416281342506409, -0.7243439555168152, -0.15906254947185516, -0.8317772150039673, -1.1626076698303223, -1.3342163562774658, -0.44414588809013367, 0.037974096834659576, -0.18865545094013214, 2.063314437866211, 0.6440221667289734, -0.8066678643226624, 2.288508415222168, 1.3564568758010864, -1.171634554862976, 1.4285507202148438, -1.152898907661438, -0.026333967223763466, 0.33161717653274536, 0.6162164211273193, -0.5590810179710388, 0.5057956576347351, 0.30211424827575684, -0.30958321690559387, 0.27419760823249817, 0.30397137999534607, 0.18316225707530975, -0.7922418117523193, -1.181965708732605, -0.13090930879116058, 0.9758108854293823, 0.7478390336036682, 2.1813831329345703, 2.349515914916992, -1.1602091789245605, -1.218732476234436, -1.103071689605713, 0.47750595211982727, -0.9114797711372375, 0.8208231925964355, -1.3531502485275269, -0.36717158555984497, -0.6692163348197937, -0.8183422088623047, 0.9733860492706299, 0.454222708940506, -1.137001872062683, -1.0501372814178467, 1.0033860206604004, 0.04216624051332474, 0.03060969151556492, -0.23344601690769196, -0.7699293494224548, 2.883819580078125, 0.508497953414917, -0.6910054087638855, 0.7385743856430054, 0.2378004491329193, 0.0003857812553178519, -0.17077453434467316, -0.5855147838592529, -0.4461878538131714, -1.7479370832443237, 1.7832345962524414, 2.2117466926574707, 1.2401021718978882, -1.005082607269287, -0.6470393538475037, 1.494524359703064, 0.8221380710601807, 0.4370344877243042, -1.1000499725341797, -1.0477491617202759, 1.255324125289917, 0.11983102560043335, -0.935347855091095, -2.0659537315368652, 1.5308603048324585, -0.41651588678359985, -1.2884187698364258, -0.05240415036678314, -1.3184007406234741, 0.5026088356971741, -1.2561591863632202, -0.2778390049934387, -2.145725727081299, -0.5164003968238831, -1.1357921361923218, -0.3676931858062744, 0.9510999917984009, 0.861781895160675, 0.3359469175338745, 1.244585394859314, -0.7295312285423279, -0.9887392520904541, 1.148601770401001, 0.0012989359674975276, -1.7451603412628174, -1.815014123916626, -0.28310659527778625, 1.8824976682662964, -0.06262164562940598, -0.8988449573516846, 0.6487563848495483, 0.6088430285453796, 1.8563889265060425, -1.7182183265686035, 2.000383138656616, 0.009843943640589714, 0.5807315707206726, 2.2699813842773438, 0.7363042831420898, 0.08203806728124619, 0.43421801924705505, -0.44385215640068054, -1.0568619966506958, -1.516505479812622, -1.4657427072525024, -0.7875732779502869, -1.8867067098617554, -0.7557993531227112, 2.044092893600464, 0.359360009431839, -2.387854814529419, -0.345485657453537, 0.15020103752613068, 0.34058114886283875, 0.625632643699646, -1.5233278274536133, 0.3228168189525604, 0.5766950845718384, 0.1084413230419159, -0.06159384548664093, -0.36275380849838257, -0.5115762948989868, 0.2390671968460083, -0.1434325873851776, -0.8470759391784668, -0.9222463965415955, -2.0242156982421875, 0.7371552586555481, 1.1880767345428467, -0.581394374370575, 1.6014389991760254, 1.100539207458496, -0.05674423649907112, -1.0440618991851807, -0.37988579273223877, -1.9199150800704956, 0.06974037736654282, 0.2909139096736908, -0.1565067023038864, 1.2828341722488403, -1.1116100549697876, -1.8374346494674683, 0.1655832976102829, -1.763810157775879, 1.458625078201294, -0.32280653715133667, 0.5343409776687622, 0.07247921079397202, 0.00240227859467268, -0.9826515316963196, -1.4164223670959473, -0.18427687883377075, -0.33124032616615295, 0.7507110834121704, 0.0648527592420578, 3.563298225402832, 0.949830949306488, -1.6927876472473145, 0.3487931787967682, -0.05120568722486496, 2.6270344257354736, -0.1337510347366333, -0.3148338496685028, -1.1405142545700073, 0.2304186224937439, 0.3848130404949188, 0.6951789259910583, -0.4257126748561859, -0.8432270288467407, 1.296546220779419, -0.21474522352218628, -0.9033794403076172, 0.6301528811454773, 1.6115275621414185, -0.6332997679710388, 0.30823302268981934, 0.4004955291748047, -2.2941722869873047, 0.7246740460395813, -1.342728614807129, -1.0246773958206177, 0.48517483472824097, 0.04159156605601311, 1.73343825340271, -0.7444857954978943, 0.9491186141967773, -1.7668023109436035, -0.984320342540741, -0.82405024766922, 1.4308844804763794, 0.49168938398361206, 0.06905114650726318, -0.40329599380493164, -0.4204418957233429, 0.3097444176673889, -0.2689228057861328, 1.4489576816558838, 0.7856079936027527, 0.9471412301063538, 0.8131369948387146, -0.792937695980072, -0.48119235038757324, -0.580651044845581, 0.9371213316917419, 2.05790114402771, -1.8183590173721313, 0.43523627519607544, 0.5625103712081909, -0.21339797973632812, -0.45415738224983215, -0.9195059537887573, 0.17652449011802673, 1.2584058046340942, 0.7772361636161804, -2.9645304679870605, -2.466975688934326, -0.2907254695892334, 1.4538545608520508, 0.2231423556804657, 0.8517555594444275, 0.3945610225200653, -0.04022981598973274, -1.5782973766326904, -0.5692254900932312, -0.6765251159667969, 1.2131667137145996, 0.40207353234291077, 0.10512244701385498, 0.2715143859386444, 0.38285601139068604, -0.07153027504682541, -0.6734410524368286, -2.2633697986602783, -1.562956690788269, -0.6956251859664917, -1.6425645351409912, -0.4042893350124359, 0.6252009272575378, 0.31482192873954773, 0.14000007510185242, 0.4763847589492798, -0.15480248630046844, 0.6526604890823364, -1.1937259435653687, 0.5352431535720825, -0.4138876497745514, -1.4445794820785522, 1.2144170999526978, 0.4054468870162964, -1.911102533340454, -1.024153709411621, 1.3246814012527466, 0.4732552170753479, -2.2607622146606445, -0.23188894987106323, 1.0528916120529175, -0.2823069095611572, -0.3435870409011841, -0.09265919029712677, 0.8146039247512817, 0.04438291862607002, 0.511604905128479, 1.0389039516448975, -0.05748613923788071, -1.5582164525985718, 0.6185095906257629, 1.544312834739685, -0.3276195824146271, 0.17975029349327087, 0.21151147782802582, -1.0759713649749756, 0.38520580530166626, -0.4685728847980499, -0.15503093600273132, -0.4414218068122864, -2.4265310764312744, -0.9510751366615295, -0.5035111308097839, 0.6238594055175781, -0.8673586249351501, -1.6202940940856934, 0.062088873237371445, -1.186166524887085, -0.4850134253501892, 0.5505208969116211, 0.9274881482124329, -0.1481567919254303, -0.8234906792640686, -0.3723835051059723, -1.845357894897461, 1.100740909576416, 0.8810153603553772, -0.002623653970658779, 0.6124134063720703, -1.4372907876968384, -0.7699332237243652, 1.3551331758499146, 0.8439428806304932, -1.1965917348861694, 0.6047784686088562, -1.3623746633529663, -0.6742733716964722, -0.2198168784379959, 0.5312091708183289, 0.4965271055698395, 1.0624805688858032, -0.5517519116401672, 1.724762201309204, 2.162310838699341, -0.11051683127880096, -1.5263540744781494, -1.0760961771011353, 0.3452169895172119, -0.3347579538822174, 0.7947940230369568, -0.7388173341751099, -0.14604303240776062, -0.8924309611320496, -0.702032208442688, -1.77492094039917, 1.6046111583709717, 0.7742415070533752, 0.32492026686668396, -1.5936734676361084, -1.4467658996582031, -0.2679276466369629, 0.08887839317321777, 0.486747682094574, -0.9904703497886658, -0.08090400695800781, -0.6087045669555664, 0.5146785974502563, 1.2874619960784912, 1.0321069955825806, 1.1789354085922241, -0.44243523478507996, 1.0979571342468262, -0.6152939796447754, -0.5412255525588989, -0.5066548585891724, 0.3310129940509796, 0.450759619474411, 0.4042391777038574, 0.777944028377533, -1.6679843664169312, -0.7540982961654663, 0.07207559049129486, 0.23821303248405457, -1.5235519409179688, -2.834602117538452, 0.8113420009613037, -0.371684193611145, -0.14082327485084534, 0.15598678588867188, -0.7522332072257996, -1.3620445728302002, -0.37569186091423035, -0.7001377940177917, 1.2189351320266724, -1.3860772848129272, 1.0451960563659668, -1.5135915279388428, -1.0728355646133423, -1.2743911743164062, 0.33978497982025146, 1.122406244277954, 0.6892522573471069, -0.6707316040992737, -0.6173789501190186, 0.3471987843513489, -0.306484580039978, -1.5812734365463257, -0.4740085303783417, -0.45739102363586426, -0.36191651225090027, -0.9897106289863586, 0.03456631675362587, 1.0068877935409546, 0.9901683330535889, -1.7632108926773071, -2.043210029602051, -0.7302117943763733, 2.3583998680114746, -0.12064532935619354, 1.317313313484192, 0.5067875385284424, -0.8583232760429382, -1.231028437614441, -0.6065053343772888, 0.18404312431812286, -0.5671141147613525, -0.17289474606513977, 1.0806645154953003, 0.6885601282119751, -0.010809788480401039, 0.06210758909583092, 0.30858495831489563, -1.0192981958389282, -0.291686475276947, 2.3683764934539795, 1.0145875215530396, 0.8986137509346008, -0.8877911567687988, -0.13251850008964539, -0.7545868158340454, 2.034396171569824, 1.5814673900604248, 1.0819729566574097, -0.32791149616241455, 0.34058713912963867, -0.693020761013031, -0.41850805282592773, -0.4396449029445648, -0.15532584488391876, 0.07502425462007523, -1.075295090675354, -1.1419671773910522, 1.3774865865707397, -0.6555546522140503, 0.12731966376304626, 0.14919936656951904, 0.41699886322021484, 0.801291823387146, 3.087733030319214, -1.967136263847351, -0.9586904048919678, 1.6990491151809692, -1.6908495426177979, -2.1583290100097656, 1.2315696477890015, 0.6144797205924988, 0.6308344602584839, -1.1030566692352295, -2.0396950244903564, -1.2315764427185059, 3.1737704277038574, 1.0972310304641724, -0.651327908039093, 2.589353084564209, -0.12645192444324493, -0.733984112739563, -0.4447003901004791, -1.6352204084396362, 0.8739778399467468, 0.16411912441253662, 0.8902482390403748, -0.5627276301383972, 1.3324308395385742, -1.6777002811431885, 1.0164928436279297, 0.8641127943992615, -0.6487624049186707, -0.7091285586357117, 0.565910816192627, -1.2900338172912598, -0.3599904179573059, -0.7019692063331604, 1.5109970569610596, -0.8248934149742126, -0.3169247806072235, -0.24441945552825928, -0.03825821354985237, 0.7952659726142883, -0.1652614176273346, -0.23448796570301056, -0.41918742656707764, 1.243301272392273, -0.5376797318458557, 1.579474925994873, -1.3879152536392212, -0.02550435997545719, 0.43106499314308167, -0.11495061963796616, 1.3409146070480347, 0.28697487711906433, -1.325626254081726, 0.8257302641868591, 0.015898263081908226, -1.0439766645431519, -0.3167405128479004, 0.9022573232650757, 1.350619912147522, 1.3031965494155884, 0.5110377669334412, 0.03950054571032524, 0.8826654553413391, 0.6325737237930298, -0.5793702006340027, -0.7923598885536194, 0.2497047483921051, -0.09346086531877518, -0.39633703231811523, -1.2053470611572266, 0.43532949686050415, -0.5452427268028259, 0.4893653690814972, -0.3822953999042511, -0.3119625151157379, 0.8756542205810547, 1.624958872795105, -0.2284456342458725, 0.007414609659463167, -1.369726538658142, 0.40484797954559326, -0.05271105840802193, 1.1864213943481445, -1.0409959554672241, 0.6146268844604492, 1.8834606409072876, 0.6883020401000977, 0.7080949544906616, -0.23436854779720306, 0.7710458040237427, -1.3887121677398682, 1.5176633596420288, 2.326855182647705, 0.04452645033597946, 0.3372933566570282, 0.28869467973709106, -0.8523188829421997, -1.365804672241211, -1.8711243867874146, 0.7854153513908386, 0.2953707277774811, 1.9025086164474487, -1.9221235513687134, -0.38497933745384216, 0.31259357929229736, -0.6277384757995605, 0.39448922872543335, -0.38517990708351135, 0.40253040194511414, -2.051766872406006, -1.1210989952087402, -1.6314711570739746, -0.26929548382759094, 0.4677882492542267, -0.7009813785552979, 0.5358653664588928, 0.7267922759056091, -0.6825883388519287, 0.17337901890277863, 0.6321101188659668, -0.30569660663604736, -1.1334599256515503, 0.7635881900787354, 2.633904218673706, -1.082214117050171, -0.5668436884880066, 0.2964678704738617, 1.0823205709457397, -0.6341980695724487, -1.0165845155715942, 1.6083532571792603, -0.047822579741477966, 0.24267315864562988, 2.807063102722168, -1.290223240852356, -1.3550776243209839, 0.6651382446289062, -1.4036744832992554, 0.42101186513900757, 0.8094975352287292, 0.39794817566871643, 1.660935401916504, 0.7019261717796326, -0.013973316177725792, 1.2127724885940552, -0.23943153023719788, -0.2233610898256302, 0.5575894713401794, -1.5181576013565063, -0.28715214133262634, -0.45258164405822754, 0.380522221326828, -0.6722116470336914, -0.46946296095848083, -0.11020923405885696, 0.49035611748695374, -1.3400346040725708, -0.5221797823905945, 0.10261697322130203, -1.3498848676681519, 1.2300807237625122, 0.6810116171836853, 1.7957606315612793, -0.05276034027338028, 0.23544643819332123, -1.2966738939285278, 2.13960337638855, 0.9595680236816406, -0.08622090518474579, 0.019455045461654663, 0.8094500303268433, 0.8753104209899902, 1.906605839729309, 0.45513179898262024, -0.6490171551704407, -1.4912357330322266, -0.4829266667366028, 0.251412570476532, -0.43936917185783386, 0.7207179069519043, 1.1647554636001587, 0.03174485266208649, -0.8073902130126953, -0.29842087626457214, -0.15358257293701172, 0.3084877133369446, -0.15620721876621246, -1.8959981203079224, -0.39409494400024414, 0.6452283263206482, -0.627346932888031, -0.3807252049446106, 2.6497926712036133, -0.31007444858551025, -2.7115371227264404, -0.5048575401306152, -0.8735126852989197, -0.9590675830841064, 0.5231752395629883, -0.3524523973464966, 0.10181336104869843, -1.3948580026626587, -0.5918375849723816, -2.262822151184082, 2.156470537185669, -1.2465171813964844, -0.5837334394454956, 0.38781803846359253, 0.5623561143875122, -1.9372867345809937, -0.35438302159309387, 0.42990803718566895, -0.3002009987831116, -0.39242637157440186, -0.8695212602615356, 0.3480885922908783, -0.5723089575767517, -0.4769570827484131, 1.754491925239563, -1.9854826927185059, -0.41845500469207764, -0.0695427879691124, 0.7170159816741943, -0.5426803231239319, -0.15873974561691284, 0.1625450849533081, -1.6427712440490723, -0.35976698994636536, 1.1381419897079468, 1.0746616125106812, 0.8566325902938843, 1.3095371723175049, 0.2988162040710449, -0.19490614533424377, -0.6808376908302307, 0.3308844566345215, 1.6061105728149414, 1.236817479133606, -0.19533294439315796, 0.1076713353395462, 0.9446232318878174, 1.06084406375885, 1.330500602722168, -0.3338034749031067, -0.6151017546653748, 0.23462006449699402, 0.5346651077270508, -0.6818456649780273, -1.1438515186309814, 0.28193625807762146, -1.5950977802276611, -2.4353907108306885, -0.29843002557754517, 1.6595933437347412, -0.4622820019721985, -0.47850871086120605, -0.4227825105190277, 0.2984481751918793, -1.1699106693267822, 0.10567401349544525, -1.0472159385681152, -1.6841188669204712, -0.37310791015625], [-0.11294993758201599, 1.5016872882843018, -1.5569384098052979, -0.06332091242074966, 0.0554133802652359, 1.7240574359893799, 1.817244291305542, 1.4335006475448608, -1.1260862350463867, -0.15105916559696198, 3.2155938148498535, 1.6254605054855347, -0.3911444842815399, -0.781146228313446, 0.43410637974739075, -0.9875398874282837, 1.0202765464782715, -0.6201590895652771, 0.8611266016960144, -1.8285696506500244, 0.9328784942626953, 0.2900281250476837, 1.707410216331482, -1.014646053314209, -0.049045808613300323, -0.7862211465835571, 0.8087397217750549, -0.13845643401145935, 1.3661569356918335, 1.684255838394165, 0.3427085876464844, 0.7393924593925476, 0.7639765739440918, 1.074670433998108, -0.8179081082344055, -1.2729685306549072, 0.7478671073913574, -0.2758277654647827, -1.001766562461853, -0.03670923411846161, -0.7025889158248901, -0.5753227472305298, 0.29576170444488525, -2.3653311729431152, 0.0991709977388382, 0.7668958306312561, -0.13582603633403778, 0.10684617608785629, -0.9430207014083862, 0.1773575395345688, 0.8785650134086609, 0.06626737117767334, 0.7994179725646973, 0.933002233505249, -0.7612165808677673, -0.06740135699510574, 0.4980207085609436, -0.1485130339860916, 0.1385861039161682, -0.20327775180339813, -0.16777074337005615, 0.8190354704856873, -0.521034300327301, -1.1536626815795898, 2.5530474185943604, 1.7250263690948486, 0.9327877163887024, -1.0466697216033936, -2.0998568534851074, -0.685957133769989, 0.5229571461677551, 0.7834556102752686, -0.6687882542610168, 0.0832347422838211, 1.0543309450149536, 0.41502445936203003, 0.0663740485906601, 1.68538236618042, -1.211496353149414, -1.3490861654281616, 0.16668039560317993, 1.3427386283874512, 1.1687078475952148, 0.35855376720428467, -0.20234571397304535, -1.1258273124694824, -0.6164645552635193, -0.3361782431602478, 0.3565721809864044, -0.6747019290924072, -0.6791474223136902, 0.557927131652832, -0.3058668076992035, -1.5166490077972412, -0.06123563274741173, -0.41388365626335144, -0.44280508160591125, 1.4303609132766724, 0.2963597774505615, -1.6049634218215942, -0.16352947056293488, 0.20739614963531494, 0.482186883687973, -1.051658034324646, 0.9139016270637512, 0.9496085047721863, -1.9995919466018677, -1.4915637969970703, -0.6397563815116882, 0.6146972179412842, -0.2695263624191284, 0.5772497057914734, 0.768519937992096, -0.5960761308670044, -1.3519046306610107, 0.8407791256904602, 0.20258431136608124, 0.6788392663002014, -0.4202357232570648, -0.8186793923377991, -0.13317088782787323, -0.24104364216327667, -0.11921615153551102, 0.3768041431903839, -1.7257896661758423, -1.0477110147476196, -0.3662351071834564, -0.7992873787879944, 0.7005767822265625, 0.7140751481056213, 1.8259481191635132, 1.44326913356781, -1.6470386981964111, 0.8437207341194153, -0.029386445879936218, -0.28822967410087585, -0.744920015335083, 0.41289836168289185, 0.5271305441856384, -0.4549640417098999, -0.5876159071922302, -1.9939687252044678, 1.3093149662017822, -0.03998443856835365, 0.06919459253549576, -1.1868267059326172, -1.057979702949524, -0.7116740345954895, -1.036529541015625, -0.13342086970806122, 0.3180595934391022, 0.32170528173446655, 0.45481234788894653, -0.9656554460525513, -0.13994990289211273, -0.051144346594810486, 0.003995598293840885, -0.276848703622818, 0.04516400769352913, -2.386679172515869, -0.8349555730819702, -0.0004781889438163489, -0.7096040844917297, -1.3220257759094238, -0.509190559387207, -1.9548219442367554, -0.3269571363925934, -0.5775059461593628, -0.6575673818588257, 1.0341880321502686, 1.2508761882781982, -0.46780216693878174, -1.3991048336029053, 0.6038889288902283, 0.7962014675140381, -1.9853712320327759, -0.653784990310669, -0.9695145487785339, 2.1755552291870117, -0.6427586674690247, 1.5768868923187256, 0.6725166440010071, 1.3221638202667236, -0.32253751158714294, 0.695327877998352, 0.6436320543289185, -0.6524344086647034, -0.33590078353881836, -1.1896487474441528, 0.5052012205123901, -2.3389458656311035, -0.7501124739646912, 0.36563462018966675, 0.44065961241722107, -0.7214556932449341, 0.9552919268608093, 0.08952955901622772, 0.655512273311615, 0.9831815361976624, -0.6653455495834351, 0.5016297101974487, -1.7170302867889404, -1.0428667068481445, 1.2984980344772339, 1.0741047859191895, 2.1831443309783936, -0.4692382514476776, -0.08406099677085876, -0.0939297303557396, -1.0629802942276, 0.5848077535629272, -0.07820317149162292, 0.5378762483596802, -0.09526724368333817, 1.0348715782165527, 0.1035667210817337, -0.143172949552536, -0.32504501938819885, -0.2998296320438385, -0.5883328914642334, 0.43580400943756104, -1.2040092945098877, 0.6574526429176331, 0.45168066024780273, 0.5213439464569092, -1.308923602104187, -0.06038028001785278, -0.9933180809020996, 0.932930052280426, -0.7105221152305603, -0.7694498896598816, 0.2853214740753174, -0.12134234607219696, -0.8893519043922424, -1.8012206554412842, -0.915924072265625, 0.5503101944923401, 0.0347757525742054, -0.6521821618080139, -1.6184197664260864, -0.3056657016277313, 0.24076202511787415, 1.2973673343658447, 1.5656354427337646, -1.0067064762115479, -0.7638749480247498, 1.0487165451049805, 0.4238404333591461, -0.5194843411445618, -0.7078648805618286, -1.531586766242981, 0.7571381330490112, -1.1508220434188843, -0.009252703748643398, -0.8307551741600037, 0.7709883451461792, 0.14813627302646637, 0.15589065849781036, -1.0210883617401123, 0.8576659560203552, -0.8891359567642212, 0.002860646927729249, -1.5042762756347656, 1.358158826828003, 0.968702495098114, -0.4805198907852173, 0.08254794031381607, -0.5731602311134338, 1.6095243692398071, -0.014022227376699448, -0.11691784113645554, 0.1461201012134552, -0.44051575660705566, -0.2448863387107849, -0.26302433013916016, 0.47125348448753357, 0.10232546925544739, -1.3578596115112305, -1.75978422164917, -1.2530014514923096, -0.3272581994533539, -0.029248028993606567, -0.9310663342475891, -0.10413903743028641, 1.1752604246139526, 0.7514644861221313, -0.7268681526184082, -1.6504814624786377, 0.10705269128084183, 2.842708110809326, 0.8277092576026917, -0.43716323375701904, -0.5140575170516968, 0.7431458234786987, -0.7322806119918823, -0.7001209855079651, -1.672414779663086, -0.2502804398536682, 0.5541940331459045, 0.24066466093063354, -1.0132958889007568, -1.2005659341812134, 1.2048346996307373, 2.0394437313079834, -0.19672048091888428, -0.15594099462032318, -2.3966851234436035, 0.7022680640220642, 0.4630788266658783, 1.7880549430847168, 0.08623840659856796, -0.8958799839019775, -0.18807265162467957, 1.4568904638290405, 0.10398590564727783, 0.13824066519737244, 1.0937821865081787, 0.5546332597732544, 0.49395355582237244, -0.4891314208507538, 0.8432779312133789, 0.3045929968357086, -0.08861952275037766, -1.0188319683074951, 1.381442666053772, 0.4536643624305725, -0.8233689665794373, 0.2561010718345642, -0.332714706659317, -0.12107150256633759, -0.1754635125398636, 2.07251238822937, 0.818844735622406, -0.4182963967323303, 1.6149795055389404, 0.3843151926994324, -2.135021209716797, -0.5053561925888062, -0.13369451463222504, 0.7428860664367676, -1.3842723369598389, 0.08880794793367386, 1.9102952480316162, 1.0803035497665405, 0.8498811721801758, -1.2814507484436035, 0.950386643409729, 0.5974435210227966, -0.0949757769703865, 0.6172016263008118, 0.03261159732937813, 1.551149845123291, 0.2565300464630127, -0.4790799915790558, 0.6019493341445923, -2.5323486328125, 0.890527606010437, 1.9034582376480103, -0.38839423656463623, 0.003434923477470875, -0.1970464140176773, -0.5614762902259827, 0.014710926450788975, 1.5724496841430664, -1.0381510257720947, -1.4914911985397339, -1.047613501548767, -1.5570505857467651, -0.9614403247833252, -0.094561368227005, 1.8061909675598145, 1.159663200378418, 0.28131425380706787, -0.9146504998207092, 1.0471019744873047, -0.12017960846424103, 0.4743921458721161, -1.4445693492889404, 0.5174800753593445, 0.6184548139572144, 1.0363788604736328, 0.3666855990886688, 0.9899292588233948, 0.15871688723564148, -0.1153831034898758, 0.5556033849716187, -0.07883309572935104, 1.3630274534225464, -0.8533170223236084, -0.01614494062960148, -0.7510843276977539, -0.34902632236480713, 0.866935670375824, -0.36092904210090637, -1.260677456855774, 0.5738793611526489, 0.23505324125289917, -0.9842745661735535, 1.9724491834640503, -0.566318690776825, 0.48875656723976135, 0.008625525049865246, 0.8355092406272888, 0.8810750842094421, -0.5710356831550598, -1.2825847864151, 0.7306497693061829, -0.6461235880851746, -1.6412079334259033, -2.161285400390625, 1.6899752616882324, 1.3248885869979858, 0.48294082283973694, 0.09542959183454514, -0.813988208770752, -1.0022016763687134, 0.2187187820672989, 1.3051912784576416, 0.923140287399292, 0.8372040390968323, -1.556792974472046, 1.3683680295944214, 0.45409631729125977, -0.9841788411140442, 1.0207520723342896, 2.010296583175659, 1.946933388710022, -1.191525936126709, 0.12144388258457184, -0.3847058415412903, 1.677195429801941, -0.06525666266679764, -0.6026773452758789, -0.17228065431118011, 0.1553896963596344, -0.8302900791168213, -1.3458906412124634, 0.36025020480155945, -1.6290380954742432, 0.9183518886566162, 0.23417431116104126, 0.23794154822826385, 1.3983896970748901, 0.4288792014122009, -1.2457090616226196, -0.8532958626747131, 1.7401264905929565, 0.7271178364753723, 0.1980314552783966, 1.1418461799621582, 1.4576762914657593, -0.19083155691623688, 0.37601813673973083, -0.22260625660419464, -1.8103649616241455, -3.0281593799591064, 1.4617379903793335, -0.6878136396408081, 1.7843877077102661, -2.1313414573669434, 1.1297353506088257, 2.7384490966796875, -0.7565467357635498, 0.23373563587665558, -0.7796242833137512, 0.04767397791147232, -0.9482946395874023, -1.8571900129318237, 0.06039995700120926, 0.7954912781715393, -0.7729319930076599, 1.3143620491027832, -1.5312530994415283, 0.3862897753715515, -1.0801112651824951, 0.5026304125785828, 0.8133854269981384, -1.5848621129989624, 0.4279917776584625, -0.576248824596405, 0.008467976935207844, -0.09487579762935638, 2.308452844619751, -0.8342383503913879, -0.4542168080806732, -0.304206520318985, -1.2202787399291992, 1.9567086696624756, -0.15225525200366974, 0.7217538356781006, -1.7488386631011963, -0.2122252732515335, 0.09164931625127792, -1.1075609922409058, 0.38315868377685547, 0.10845569521188736, -1.4854000806808472, 0.9723572134971619, -0.4380717873573303, 0.7899146676063538, 0.3178020715713501, 0.4178259074687958, 0.7108185887336731, 1.7293399572372437, 0.5223357677459717, 0.7960121631622314, -0.9197636246681213, -0.7194663286209106, 1.4351952075958252, 0.7083175778388977, -1.7628153562545776, 0.9686563014984131, -0.6364453434944153, 0.6932663321495056, 1.0115070343017578, -0.19261635839939117, 0.5670584440231323, -1.2659229040145874, -0.3728887736797333, -0.41530683636665344, 0.10583879798650742, -0.5033165812492371, -1.1701397895812988, 1.3310825824737549, 0.11083976924419403, -1.3167635202407837, 2.617859363555908, -0.47602495551109314, -0.822422444820404, 1.304918885231018, -1.2999851703643799, -0.887698769569397, -0.725159764289856, 0.749086856842041, -0.39542320370674133, -0.14151355624198914, -2.060767412185669, 0.3990132212638855, 0.15459652245044708, -0.6434407830238342, 0.5917026400566101, 0.9288092851638794, 1.1518789529800415, -1.5077553987503052, 0.6674280166625977, 2.2548470497131348, -0.3592239022254944, -0.6954652667045593, -0.19648176431655884, -1.0615947246551514, 1.1506835222244263, 0.6308456659317017, 2.0116541385650635, -0.35697075724601746, -0.18446899950504303, 0.16862830519676208, -1.2299748659133911, 0.2621655762195587, -1.188506841659546, -0.2963516116142273, 0.8089778423309326, 0.1366206258535385, 0.775034487247467, -1.1565546989440918, -0.40124234557151794, -1.2941137552261353, -0.3726158142089844, -0.2086125910282135, 0.6616613268852234, -0.42989587783813477, -0.37607550621032715, -0.2518135607242584, -0.8244867324829102, 0.2844659984111786, -0.43555834889411926, -1.681104302406311, 0.7066280841827393, 0.22794048488140106, 1.4414403438568115, 0.5683965086936951, -1.1068216562271118, 1.0641433000564575, -2.694533586502075, 0.16583208739757538, 0.7456055879592896, -0.3519720137119293, 0.15752941370010376, -1.557235598564148, -0.33581310510635376, 0.3904436230659485, -0.4694134294986725, -0.06800797581672668, -0.5793206095695496, -0.06994400173425674, 1.0885459184646606, -0.6118496060371399, 1.2366031408309937, 0.0822744220495224, -0.07052911072969437, 0.6485756039619446, 1.3115166425704956, 0.5755025744438171, -0.47775134444236755, -0.6518181562423706, -1.1798397302627563, 1.910982608795166, -0.5838778018951416, -0.8370386958122253, 0.46694278717041016, 0.4839010536670685, 1.0025559663772583, 1.1668211221694946, -1.5279299020767212, 1.2453728914260864, -1.0903639793395996, 1.036863088607788, -0.34084081649780273, 0.2250257283449173, -0.5477524399757385, -0.12080924212932587, -0.9417667984962463, 0.08330743759870529, 1.0047798156738281, -0.006559513974934816, -1.5431814193725586, 0.7628589272499084, 0.0981869325041771, 0.533287525177002, 0.562391996383667, 0.31462225317955017, -0.5623667240142822, 0.32490283250808716, 0.5011147260665894, 0.9540308117866516, -0.10108482092618942, -1.0221225023269653, 1.7369346618652344, 1.3488627672195435, -0.17356035113334656, 3.0140111446380615, -0.8578866124153137, -1.5688849687576294, 0.711506724357605, 0.38983824849128723, 0.7501474618911743, 1.9755651950836182, 0.12367844581604004, 0.2970261871814728, -0.30914101004600525, 1.2589048147201538, 0.010770917870104313, 0.6505719423294067, 0.909869372844696, -1.6717945337295532, 0.7017760872840881, 0.2631687819957733, 2.066472291946411, -0.33636578917503357, 0.902146577835083, 0.26850220561027527, -1.192186713218689, -0.5218479633331299, 1.649485468864441, -1.06048583984375, -0.4264789819717407, 1.5862222909927368, -2.2255513668060303, -1.3067318201065063, -0.25600484013557434, 1.2563016414642334, -2.290797710418701, -1.018538236618042, -1.2047263383865356, -1.6912115812301636, -1.036863088607788, 1.7973803281784058, 0.6871635913848877, 0.23915651440620422, 0.1811959594488144, -1.8825175762176514, -0.7858487963676453, -1.2904843091964722, -0.5032013654708862, -1.341043472290039, -1.6197562217712402, -0.7272545695304871, 0.8484418392181396, 0.9283809661865234, 0.3860040605068207, 0.6363003253936768, -0.2280847281217575, 1.7586228847503662, 1.0209691524505615, 1.8335715532302856, 0.34192904829978943, 0.27075666189193726, -1.5049656629562378, 0.8648931980133057, -1.2477365732192993, -0.6454417705535889, -0.19006940722465515, -0.7814934849739075, 0.926879346370697, -0.12671571969985962, -1.1090741157531738, 1.3095349073410034, -0.6505703330039978, -0.2630775272846222, 1.4689581394195557, 0.1161932572722435, 1.1769404411315918, 1.7771846055984497, 1.4838130474090576, 0.43447425961494446, 0.2059205323457718, 0.9841245412826538, -0.5013439655303955, 0.08484815061092377, 1.5882457494735718, 0.6942358613014221, 0.8744193315505981, -0.5601195693016052, -0.35173919796943665, -2.3475799560546875, 1.229046106338501, 1.2342643737792969, 0.7767070531845093, -1.4913909435272217, 0.33965328335762024, 1.6807154417037964, -0.3146990239620209, 0.5839623212814331, -0.20747512578964233, 0.3123409152030945, 0.2743520140647888, -0.06353042274713516, -1.1337568759918213, 0.6076449155807495, -0.34570562839508057, 0.7355226874351501, 0.4321286082267761, 0.28177958726882935, -0.25079023838043213, 0.7118085026741028, -2.2047157287597656, -1.1806689500808716, 0.8340710401535034, 0.09810991585254669, 0.5057129859924316, 1.540401816368103, 0.7136369347572327, -1.3451181650161743, 0.34612223505973816, -0.705748438835144, -1.830642580986023, 0.7921937108039856, 1.2689977884292603, 1.1398954391479492, 2.0905792713165283, -0.538127064704895, -0.2322266399860382, 0.41054850816726685, -0.015103469602763653, -0.2324559986591339, 0.259591668844223, 0.6100406050682068, 1.2558308839797974, -0.358769953250885, 0.14323392510414124, 1.4904165267944336, 0.05966435745358467, -0.9364458918571472, 1.1530667543411255, 0.13803648948669434, 1.849117636680603, 1.377917766571045, -0.8752222061157227, 0.8383851647377014, 0.39399364590644836, 0.47437381744384766, 1.458461880683899, -0.27394193410873413, -0.09934087097644806, -1.1215381622314453, -0.08458264172077179, 0.4205075204372406, 0.07508087903261185, -1.1281366348266602, -0.4799598455429077, -2.6924729347229004, 0.9171205163002014, -0.556195855140686, 0.7716796398162842, -0.589608371257782, -0.721470057964325, 1.7573795318603516, 0.04891211539506912, 2.338364362716675, 0.6240823268890381, -0.9215217232704163, -0.3671850562095642, 0.09364904463291168, -1.907318353652954, -1.0797979831695557, 1.7528631687164307, 1.27618408203125, -0.4690597355365753, 0.6435851454734802, 1.7241528034210205, -1.2294820547103882, 0.29648348689079285, 1.0383546352386475, -0.07750599086284637, 0.7704665064811707, 0.9282940030097961, -1.5538727045059204, 1.0284162759780884, 1.263260006904602, 0.24031466245651245, -1.3415066003799438, 0.44855907559394836, -0.4814506769180298, 0.9866454601287842, 0.28834059834480286, 1.1126511096954346, 1.350921630859375, -0.4852193593978882, -0.1380699723958969, 0.18598362803459167, -1.0382897853851318, -0.8868938684463501, -0.42145195603370667, 1.622083306312561, 0.9782228469848633, 1.5666061639785767, -2.3014678955078125, 0.10071117430925369, 1.2496060132980347, 0.5529727339744568, -1.7267848253250122, 0.3862934112548828, 0.34255465865135193, 2.481147050857544, -0.03191803768277168, -2.4534785747528076, -0.2891044020652771, -0.5157498121261597, 1.4799367189407349, -0.22204825282096863, -0.5719646215438843, -0.2737775146961212, -0.6374572515487671, 1.9322909116744995, -0.034715138375759125, 0.6586096286773682, 0.6063557863235474, 1.0808720588684082, -0.911363959312439, 0.608424961566925, 1.9794782400131226, -0.35513314604759216, 0.03177337348461151, -0.24576327204704285, -1.5609474182128906, 0.07532462477684021, 1.826335072517395, -0.9749610424041748, -0.4069940447807312, 0.8444125056266785, -0.38245120644569397, -0.1157383844256401, -0.7863340973854065, 1.0368752479553223, -1.0021882057189941, -0.8868524432182312, -0.5149103999137878, 1.3784652948379517, 1.878701090812683, -0.16797253489494324, 0.8601852059364319, 2.5827383995056152, 1.214883804321289, 0.5528185963630676, -1.071024775505066, -1.0935488939285278, -0.09742359071969986, 0.7226366400718689, 0.753433346748352, 0.6489998698234558, -1.3269141912460327, -0.5559860467910767, 0.9026747941970825, 1.847939372062683, -0.5997872948646545, 0.20651446282863617, 1.3144304752349854, 1.0922012329101562, 0.264759361743927, -0.7171360850334167, -0.004576528910547495, 0.13683414459228516, -1.5669602155685425, -2.1762242317199707, -0.326617568731308, -1.3346483707427979, -0.8993481397628784, -0.08944518119096756, 0.5200627446174622, -0.574501633644104, 0.2835707366466522, 1.2602039575576782, 0.386200487613678, 0.41911551356315613, -0.7378063797950745, 0.2028297781944275, 1.2370309829711914, -1.3371933698654175, 0.09444739669561386, 0.364641934633255, -0.17873765528202057, 0.5096559524536133, -1.8402833938598633, -1.189488410949707, 2.167576313018799, -0.30452844500541687, -1.0174833536148071, 0.4608880281448364, 0.9778062105178833, 0.4891843795776367, 1.2553244829177856, 0.9295943975448608, 1.1281052827835083, 0.15780286490917206, -0.9241790175437927, -0.495524138212204, 1.3890223503112793, -0.04685007780790329, 0.46792763471603394, -0.1703261286020279, 1.387450933456421, 0.5502406358718872, -2.067652940750122, -1.148400902748108, -0.8921011090278625, 1.1618155241012573, 1.6713365316390991, -0.4224424660205841, 0.04649841785430908, 0.463575541973114, -0.08872223645448685, -0.7259487509727478, -0.5107312798500061, 2.2371718883514404, -0.5495654344558716, -0.5428544282913208, 0.9850358963012695, 0.6661985516548157, 0.35585588216781616, -0.4162273108959198, -0.632625162601471, 0.9105641841888428, -0.12171360850334167, 1.0821584463119507, 0.00039026146987453103, 1.0364190340042114, -0.5021300315856934, 0.6839936375617981, 0.25070905685424805, 1.5640913248062134, 1.5301566123962402, 0.5457803010940552, 0.518609881401062, 0.0025572918821126223, -0.920051634311676, 0.5226525068283081, -0.6019607186317444, -2.018411874771118, -1.1955890655517578, 0.536644458770752, -1.3477920293807983, -1.0384467840194702, 1.5973769426345825, -0.5827511548995972, -0.9586367011070251, 0.856432318687439, 0.6474496126174927, -0.35612916946411133, -0.684002697467804, -0.640083372592926, -1.0240933895111084, 1.1350356340408325, 0.6382095217704773, 0.4162405729293823, 2.903069019317627, 0.18538792431354523, -0.7199545502662659, 1.043853998184204, -1.9387528896331787, 0.025453481823205948, 0.451205849647522], [1.0444865226745605, 0.2001810222864151, 0.8677982091903687, -1.2258752584457397, 0.3305976986885071, 0.6916252374649048, -1.5128178596496582, 0.05040311813354492, -0.6119596362113953, -0.004065188579261303, -0.28125864267349243, -1.353285551071167, -0.30672019720077515, 0.3294171690940857, -0.7563528418540955, -0.4417758882045746, -0.3730677664279938, 0.9236100912094116, 0.7939465641975403, -1.1596095561981201, -0.03551016375422478, 0.1489894539117813, -0.23167604207992554, 0.039582252502441406, -0.9694807529449463, -0.2913375198841095, 0.8576592206954956, -0.39089834690093994, 0.2086479663848877, 0.738278865814209, 0.6081814765930176, 1.726683497428894, -0.4169676899909973, 0.12993383407592773, -1.10029137134552, -0.6305108666419983, 1.2446571588516235, -0.7890781760215759, 2.5220906734466553, 0.22886155545711517, 0.4459245502948761, -0.8366793990135193, -0.5561423301696777, -0.9123261570930481, -0.8632451295852661, 0.446188360452652, -0.7774487733840942, 2.7997756004333496, 0.4499501585960388, -0.21370014548301697, 0.9596097469329834, -0.7746561765670776, 1.0693519115447998, 1.2830722332000732, -0.27606329321861267, -0.0825953483581543, -2.387059450149536, 1.2728887796401978, 0.3665566146373749, 0.8389058709144592, -0.8234984874725342, 0.7866398096084595, 1.2587435245513916, 0.3172808289527893, 0.1914137303829193, 1.3412957191467285, -0.35898661613464355, 0.3322286307811737, -0.7395948767662048, -0.34829819202423096, 0.2906559109687805, 0.7327268123626709, -0.43339914083480835, -0.032088130712509155, -0.9819590449333191, -0.6183879971504211, -0.2813457250595093, 0.019394321367144585, -0.0032459774520248175, -0.4133109748363495, 0.9081853032112122, -0.45726868510246277, -0.584888219833374, 0.7654758095741272, -0.6389152407646179, -0.9001184105873108, -1.4736191034317017, -0.8791859745979309, -0.4903890788555145, 1.0700184106826782, 2.0850980281829834, 2.6727638244628906, 0.3668134808540344, -1.3328737020492554, -1.975158452987671, 1.872833251953125, -1.8000736236572266, -1.2157427072525024, -1.076509952545166, 0.8232683539390564, 0.10612107068300247, -1.9528816938400269, -0.393399178981781, -0.5055912733078003, 0.5888087749481201, 0.05895645543932915, -1.874473214149475, -0.18908511102199554, 0.8319835066795349, -0.0227140374481678, -0.1145421713590622, 0.16763941943645477, -0.8132068514823914, 0.6073718070983887, -0.3210306167602539, -1.6400835514068604, 0.7457373738288879, 0.8078919649124146, -1.5524039268493652, -0.29919424653053284, 0.3256857991218567, -0.32016968727111816, 0.9807774424552917, 1.763764500617981, -0.14336107671260834, 0.2381313443183899, 0.1929374486207962, -0.59837806224823, 0.4647541344165802, -0.8200318813323975, -0.6082965135574341, -1.1691399812698364, -0.21548520028591156, 1.2865101099014282, -1.249795913696289, -1.543552279472351, -0.2839665412902832, 0.18454323709011078, 0.3528023064136505, -2.187344551086426, 1.154404878616333, 0.8393645882606506, -1.3790169954299927, -0.38396984338760376, 0.9257561564445496, 1.5330836772918701, 0.031803201884031296, 0.11724888533353806, 0.40849265456199646, 0.6744706630706787, 0.9322047829627991, 0.21576137840747833, -2.0696938037872314, -0.0740073025226593, -1.8670004606246948, -2.2225489616394043, 2.154747486114502, 0.7767680287361145, 1.355428695678711, -0.5032809376716614, 1.3775016069412231, 0.4537794888019562, -0.07391732931137085, -0.0027761515229940414, -2.5370566844940186, 0.5079435706138611, -0.3319561183452606, -0.38803938031196594, 0.021959858015179634, 0.39355191588401794, -0.9190229773521423, 0.14420592784881592, -0.8836538195610046, -0.018996719270944595, 0.34692028164863586, 0.6577188372612, -1.8649884462356567, 0.7496073842048645, -0.23683977127075195, -0.43828853964805603, 0.745760977268219, 0.2134813368320465, 0.9349930882453918, -0.7231748700141907, -0.8711258769035339, -0.06866888701915741, -1.0149362087249756, 0.3288643956184387, 0.8170731067657471, -0.8747732043266296, -0.373084157705307, 1.2636908292770386, -0.13157138228416443, -0.337820440530777, 1.0569486618041992, -0.6279898881912231, 0.9932927489280701, -1.1451181173324585, -2.1122334003448486, -0.4192636013031006, 0.5147877931594849, 1.828912377357483, 0.02068641223013401, -0.7683727741241455, -0.07290423661470413, -0.3093319237232208, -0.26388436555862427, -0.513763964176178, -0.36411261558532715, -1.2846505641937256, 0.2885182201862335, 0.35197004675865173, -0.761770486831665, -0.1736295521259308, -0.39090949296951294, -0.9428280591964722, -1.359370231628418, -0.5196812152862549, -0.5272858142852783, 1.1005467176437378, 0.17424888908863068, 1.3651138544082642, -0.8838122487068176, 0.05320873111486435, -0.49330997467041016, 0.39334559440612793, -0.41013064980506897, 0.744485080242157, -1.6125855445861816, 0.14383380115032196, 0.4464554786682129, 0.8242036700248718, 0.8512179255485535, 0.8740065097808838, -0.9134808778762817, -0.16869518160820007, 0.5192636251449585, -1.023606300354004, 2.190321922302246, 1.14448082447052, 0.5350333452224731, -0.9667453169822693, -1.0580638647079468, -2.2010903358459473, 0.07798468321561813, 1.8597149848937988, -1.3805193901062012, 1.0879616737365723, -0.8143323659896851, 1.9490737915039062, 0.21196547150611877, -1.3072513341903687, -0.8261590600013733, 1.6460539102554321, 1.095367431640625, 0.09473760426044464, 1.4668688774108887, -0.005540103185921907, -0.3776092529296875, -0.0564529225230217, -2.1081724166870117, -0.6760526299476624, -0.11911032348871231, 0.02380002662539482, -1.1922467947006226, 1.8536449670791626, -0.8499791622161865, -1.2753762006759644, 0.6010007262229919, -0.5244473814964294, 0.016032187268137932, 0.15967923402786255, 0.0355568565428257, -2.1901443004608154, 0.2843003273010254, -0.5774217844009399, 0.1639580875635147, 1.2886680364608765, -0.1921708881855011, 0.5237479209899902, 1.2354230880737305, 1.3563247919082642, 1.365385890007019, -1.0583962202072144, 0.27658501267433167, 0.8486860990524292, -3.467604637145996, -0.5924422144889832, -1.9868556261062622, 2.156107187271118, 0.4803948402404785, -1.826351523399353, 0.02019844576716423, 0.6968889832496643, 0.023632800206542015, -0.302543044090271, -0.32653045654296875, 0.5859342813491821, -0.8288165330886841, 0.2267707884311676, 2.226658821105957, -1.605089783668518, 0.08036268502473831, 1.9836400747299194, -0.0530085563659668, -1.310105800628662, 0.36093443632125854, 0.11409059911966324, -1.73443603515625, -1.0760666131973267, -1.4950202703475952, -0.5012762546539307, 2.1305816173553467, 1.3989518880844116, -1.1373882293701172, -0.22216686606407166, -2.3960001468658447, -0.19529388844966888, -0.6102783679962158, 0.08616728335618973, 1.2731072902679443, -0.1745033860206604, 0.16704805195331573, -0.10755216330289841, 0.013703415170311928, 1.8950449228286743, 0.3510226011276245, 0.048544567078351974, -0.11732892692089081, -0.5809670686721802, -0.01846781000494957, -0.34581202268600464, 0.6244943737983704, 0.0072934431955218315, -0.854189932346344, 0.7242521047592163, -2.129606246948242, 0.5472646355628967, -0.34681475162506104, -0.7433812022209167, 1.0443171262741089, 0.9320998787879944, -0.08689213544130325, -0.22881434857845306, -1.0376205444335938, 0.846889078617096, -2.121450901031494, -0.5761076211929321, -2.379081964492798, -0.12556162476539612, -0.6995493173599243, 1.5498688220977783, 0.18638651072978973, 0.30179670453071594, 0.804990291595459, -1.1689027547836304, 1.1097909212112427, 1.3019022941589355, 0.2532607614994049, 0.9389157891273499, 0.07110065221786499, -2.488090991973877, 1.3930020332336426, 0.7555659413337708, -1.2545355558395386, -0.1253974735736847, 0.02849642001092434, -1.2646883726119995, 1.2747331857681274, -0.6356675028800964, 1.351024866104126, 0.4419185519218445, 1.3215892314910889, -0.4457889795303345, 1.3230561017990112, 0.2882055640220642, 0.8752288222312927, -0.018561724573373795, -0.44052669405937195, 1.1700682640075684, 0.4812484681606293, -0.7489339709281921, -0.1536409854888916, 0.9044121503829956, 0.1388629823923111, -0.31486931443214417, 0.6563006043434143, -1.726969838142395, -0.7915902137756348, -1.1388881206512451, -0.9209718704223633, -0.20167399942874908, 1.3545230627059937, -0.5080669522285461, -1.0542163848876953, -0.9672793745994568, -0.5869001150131226, -0.5214324593544006, 0.7549601197242737, -1.4764807224273682, -0.550322949886322, 0.13942185044288635, 1.2102090120315552, 2.091555595397949, -0.4610937535762787, 0.35064610838890076, -0.3627181947231293, -1.5216481685638428, 1.055195689201355, 0.1891162097454071, -0.9747977256774902, -0.8400007486343384, -1.1813147068023682, -0.10593412816524506, 1.4403115510940552, 0.6467636823654175, -0.6159198880195618, -0.9748803973197937, -0.2895357012748718, -0.3510175049304962, -0.9283972978591919, -2.2037129402160645, -0.3960105776786804, 0.7686980962753296, 1.6738855838775635, -0.42049428820610046, -0.03958633914589882, 0.7715886831283569, -0.5561782121658325, 0.7823039889335632, -1.5745174884796143, -1.2202110290527344, -0.8752236366271973, 0.062218137085437775, -0.30849355459213257, 0.328181654214859, -0.03964166343212128, 1.0172048807144165, 1.3897860050201416, 0.6965205669403076, -0.41320106387138367, 0.6417480111122131, 0.8038228154182434, 0.048151396214962006, -0.12069276720285416, 0.4979976713657379, 0.29296064376831055, -1.7260934114456177, -1.397423505783081, -1.8475488424301147, 0.6875718235969543, 0.7161420583724976, -0.6102203130722046, 0.6980676054954529, 0.17975136637687683, 0.5254415273666382, 0.5096674561500549, -0.11407601833343506, 0.6303573250770569, -0.1585003137588501, 0.4303087592124939, 0.8165830969810486, -0.31781721115112305, -0.09671612083911896, -0.35507625341415405, -0.649226725101471, 0.13535068929195404, -1.7858892679214478, -2.0493874549865723, 1.7938439846038818, -0.9502183198928833, 0.7404763698577881, -0.9561804533004761, -0.22176308929920197, 2.4788551330566406, -1.9021482467651367, 0.8061060905456543, 0.5315218567848206, 0.8200269341468811, -0.22452548146247864, 0.18059778213500977, 0.2501384913921356, -0.8308793902397156, 0.9407126307487488, -1.0608420372009277, -2.222834348678589, 0.4422459006309509, -0.5792306661605835, 1.3595319986343384, -0.15013886988162994, -2.015510320663452, -0.7601553201675415, 0.46063899993896484, -1.2026278972625732, -1.0211831331253052, -0.10145614296197891, 1.6367931365966797, -1.2364848852157593, 1.2654300928115845, 0.8289097547531128, 0.2386673390865326, -0.9513953328132629, 1.4042999744415283, 0.4677097201347351, 1.14752197265625, 0.21006512641906738, 0.013359091244637966, 0.20079956948757172, -0.652015209197998, -1.2317901849746704, -0.7738243937492371, -2.149587869644165, 1.9284344911575317, 0.2781268358230591, -1.5921083688735962, 0.29261305928230286, -0.7054240703582764, 0.9347184896469116, 1.2246809005737305, -1.1098086833953857, 0.8649235367774963, 1.2565076351165771, 1.3990942239761353, 0.3965991139411926, -0.42187973856925964, -0.35499435663223267, -0.7307186126708984, -0.4291057288646698, -0.9675933122634888, 0.447100430727005, -1.3265180587768555, -2.6558127403259277, -1.3451647758483887, -0.038454569876194, 0.539569616317749, -0.2072206735610962, 0.6081919074058533, -0.4204123318195343, 0.7286632061004639, -0.7176520228385925, -0.7419347167015076, -0.15919972956180573, -1.1769062280654907, -0.4954063594341278, -0.20434680581092834, 0.5588781833648682, -0.16039647161960602, 0.35775262117385864, -1.006560206413269, -0.6368533968925476, -0.24407000839710236, -1.6096160411834717, 1.5302581787109375, -2.0818238258361816, -0.07056284695863724, -0.413961797952652, -0.5289332270622253, -0.23131144046783447, 1.2929526567459106, 0.6182065606117249, -0.4897776246070862, -0.1393800973892212, 1.8085201978683472, 0.6220486760139465, 1.032950758934021, 1.3460092544555664, -1.2928622961044312, 1.186232089996338, 1.2781156301498413, -1.766940951347351, 0.9406410455703735, 0.6955577731132507, -0.7860601544380188, -1.5319355726242065, 1.064923882484436, 0.7014485597610474, -0.20528803765773773, 1.0955358743667603, 1.308656930923462, -0.8783389925956726, 0.6448330283164978, 0.7327595949172974, 0.9820383787155151, -1.5926709175109863, -0.18976998329162598, -2.1149697303771973, 0.11593158543109894, -0.42151758074760437, 0.05774121358990669, -1.5339505672454834, 0.36580023169517517, -0.35113903880119324, -0.30835652351379395, 1.448643684387207, -0.5980699062347412, 1.8098667860031128, 0.4675021469593048, -0.28778889775276184, -1.4930334091186523, -0.6240761876106262, 0.7440525889396667, -0.13558782637119293, -0.3464633822441101, -1.5475008487701416, -1.2276469469070435, 0.18657661974430084, 1.367075800895691, -0.3081885576248169, -1.3142844438552856, 0.35740432143211365, 1.043432593345642, 1.397399663925171, 0.27179625630378723, -2.0742526054382324, -1.8981562852859497, 2.838268280029297, 0.683628261089325, 0.41719913482666016, -0.4127011001110077, 0.3952673077583313, -0.5046959519386292, -0.7047301530838013, -1.6730629205703735, -0.7127963304519653, 0.5007957220077515, -0.6949547529220581, 0.6166484355926514, 0.14676032960414886, -0.23575738072395325, 0.11054803431034088, -0.031940020620822906, -0.022563129663467407, -1.7154089212417603, -0.3411325216293335, 0.7000875473022461, 2.0129880905151367, 1.0699135065078735, -1.967200756072998, -0.15714780986309052, -0.8753332495689392, -1.0806586742401123, 0.8091142773628235, -0.2833382785320282, -0.6733196377754211, 1.8472988605499268, 2.2328968048095703, 0.14081119000911713, -0.5095331072807312, -1.8094154596328735, 1.0969955921173096, -0.39383187890052795, 0.8384184837341309, 0.7005000710487366, -0.6042508482933044, 0.14003023505210876, 0.854315996170044, -0.2955067455768585, 2.0858957767486572, 0.16115006804466248, 0.23296497762203217, 1.2026854753494263, -0.8811586499214172, -0.6491857171058655, -0.6035438179969788, 0.25287091732025146, -1.0871864557266235, -1.1090511083602905, -0.8663045763969421, 2.2877037525177, -1.360094428062439, -0.13344749808311462, 0.017439477145671844, -1.8361090421676636, -1.4295333623886108, 0.8745023608207703, -1.740095615386963, -2.8446156978607178, 0.18584004044532776, 1.076222538948059, 1.4809939861297607, 1.044053077697754, -0.06772755831480026, 0.731387197971344, 0.0012689536670222878, -1.0019729137420654, -1.3828635215759277, 1.4857234954833984, 2.372230291366577, -1.1484885215759277, 0.21108564734458923, 0.23627126216888428, -2.4533677101135254, 1.5296213626861572, -1.2025896310806274, 0.5845800042152405, 0.14016835391521454, -1.0278605222702026, -0.4816436171531677, -0.1831725686788559, -0.45607703924179077, 0.41328155994415283, 0.036625321954488754, -1.212277889251709, -0.31561392545700073, -1.64320707321167, 0.49899548292160034, 0.8642928600311279, -0.6844402551651001, -0.10132422298192978, -0.5059205889701843, 1.1837739944458008, 0.8151512145996094, -0.3458191454410553, 0.5330387949943542, -0.15408895909786224, -0.4135763347148895, 0.5008264183998108, -0.8057782649993896, 0.5142449140548706, 1.0426764488220215, 0.5127596259117126, -1.2017489671707153, 0.7229970693588257, 0.8211948275566101, 0.9599484801292419, 0.5496016144752502, -1.062652826309204, 0.2879432737827301, 0.12725333869457245, 0.1714269369840622, -0.15207624435424805, 0.007268228568136692, -2.047072649002075, 0.10115447640419006, 0.2269585132598877, -1.0355783700942993, -1.2654038667678833, 0.28528833389282227, 0.41993096470832825, -0.2060178965330124, -0.6502509713172913, 1.5688457489013672, 1.1230862140655518, 0.5951850414276123, 0.8497157692909241, -0.5868121385574341, 0.8080899119377136, 0.38101017475128174, -0.8768452405929565, -0.21919018030166626, 0.26526790857315063, -1.215962290763855, 0.2394786775112152, 0.06995191425085068, 0.266370564699173, -0.5412341952323914, 1.1735153198242188, 2.8225724697113037, 0.7196919322013855, 0.20374520123004913, -0.7841286063194275, 0.3705730438232422, -0.16245409846305847, 0.10330215841531754, 1.1738697290420532, -0.24721047282218933, -0.12085007131099701, -0.4139300286769867, 0.31758585572242737, 0.3470650315284729, -0.47694674134254456, -0.18511086702346802, 0.5330585241317749, -1.499718189239502, -0.855362057685852, 1.7436904907226562, 2.445511817932129, -1.388648271560669, -1.33440101146698, -1.631497859954834, -0.982694685459137, 0.15229591727256775, 0.018442116677761078, 0.7919120192527771, -0.29383641481399536, -1.2623523473739624, 0.21276652812957764, 1.8547102212905884, 0.40489745140075684, -0.36171984672546387, -1.111391544342041, 1.7399797439575195, 1.70579195022583, -0.3597234785556793, -1.1644713878631592, 0.13186873495578766, 0.4068291485309601, 0.5515672564506531, -0.2844773828983307, 1.3498626947402954, 0.4398581385612488, 0.564218282699585, -0.3484295606613159, 0.21691463887691498, -0.5585595965385437, -1.6047255992889404, 1.3881269693374634, 1.2066398859024048, -0.3715479373931885, 1.6871544122695923, -0.35336989164352417, 0.051080476492643356, -0.024962861090898514, -0.6507844924926758, 1.8332369327545166, -0.067175954580307, 1.3004206418991089, -1.0599461793899536, -1.2316213846206665, 1.1888339519500732, -0.4319838583469391, 0.8091236352920532, -1.7251646518707275, 1.7963933944702148, 1.6185619831085205, 0.8129748106002808, -0.12960337102413177, -1.8326106071472168, -0.6797130107879639, 0.8956472277641296, 1.758044958114624, -0.35186144709587097, 0.008265255019068718, 0.3679424226284027, -0.08503264933824539, 2.0626440048217773, -0.0037014561239629984, 1.5316542387008667, -0.2714602053165436, 1.1085506677627563, 0.33795514702796936, -0.9392845034599304, 0.3358834981918335, 2.2750375270843506, 0.6724454164505005, -0.40251556038856506, -0.23360300064086914, 0.2410593181848526, -0.15750686824321747, -0.6917137503623962, -1.161821961402893, -1.2548906803131104, 2.480722427368164, 0.2739003002643585, -2.3441336154937744, -1.4346213340759277, 0.5810766220092773, -1.9163459539413452, -0.5919109582901001, 0.170929953455925, 0.2760136127471924, -0.3984447121620178, -0.1184661015868187, -0.9499393701553345, 1.2105165719985962, -1.626306176185608, -0.9348533749580383, -0.7881023287773132, 0.3968943953514099, 0.3457278609275818, -1.2533220052719116, -0.7878350615501404, -1.3115222454071045, 0.1591726541519165, -0.38672012090682983, 0.6603729128837585, -0.7048208117485046, -0.24566389620304108, 0.09780246019363403, 1.8303014039993286, 0.39419567584991455, 0.7767348289489746, -0.37408143281936646, 0.5416340231895447, -0.9731485843658447, 0.48466479778289795, 0.20590521395206451, 0.5483015775680542, 0.23035088181495667, -0.560158371925354, 0.5837802886962891, 0.3141638934612274, -0.8258086442947388, -1.4860731363296509, 0.33076027035713196, -0.457311749458313, 0.05944797396659851, 0.555932879447937, -0.3585776090621948, -0.40757548809051514, -0.23279564082622528, -1.5462478399276733, 0.8542478084564209, -0.07099601626396179, 0.8178693056106567, 0.29092955589294434, -0.3497565984725952, 1.1652796268463135, 0.4701234996318817, 0.46837183833122253, -0.4853198230266571, 0.52093505859375, -1.4238739013671875, 1.4190378189086914, 2.5224626064300537, -1.0963730812072754, 0.15621578693389893, 0.6982500553131104, -1.8039746284484863, 0.20225924253463745, 0.33694055676460266, 0.19296976923942566, 0.09625782817602158, 0.24697963893413544, -0.6870889663696289, -0.8146517872810364, 0.7177279591560364, 0.786690354347229, 0.21649393439292908, -0.7802314758300781, 0.28943026065826416, 0.5572245717048645, 0.954452395439148, 0.7735992670059204, -1.314761996269226, -0.93638014793396, 0.049034491181373596, -0.27890077233314514, 1.5404211282730103, 0.07808845490217209, 0.0868656262755394, 0.05312669277191162, 0.6456570625305176, 0.027791501954197884, -2.3148481845855713, 2.0349485874176025, 0.23966829478740692, 0.42835018038749695, -0.6051558256149292, -0.0038612426724284887, 0.4194839894771576, -0.8096935749053955, 0.9061692357063293, 0.34585434198379517, -3.2650675773620605, -0.6528272032737732, -0.1539628654718399, -0.4163820445537567, 0.7206942439079285, -0.3033151626586914, -0.1901128888130188, -0.8473642468452454, -0.3509562313556671, -0.05822982266545296, 0.31801092624664307, 2.025122880935669, 1.4632713794708252, 1.0967451333999634, 0.5562528967857361, 0.6618943810462952, -0.12408585101366043, 2.8831145763397217, 0.2890775501728058, -0.522932231426239, 1.0620992183685303, -0.3692452013492584, 1.0750157833099365, 0.6828508377075195, 1.5610642433166504, -1.4352010488510132, -0.26361480355262756, 0.11447491496801376, -0.052481915801763535, 0.4994382858276367, 0.7039594650268555, 0.2937205731868744, 0.7063030004501343, -1.4193435907363892, -0.4110919237136841, 1.7407938241958618, 0.027225598692893982, -1.0186245441436768, 1.0310698747634888, -0.7134324908256531, -0.9132281541824341, 0.962261438369751, 0.8858211636543274, -0.2538401484489441, 1.7809805870056152, 0.21704646944999695], [0.49769166111946106, 0.14644575119018555, 0.8683226704597473, 0.5347753763198853, 0.6045018434524536, 0.9449782967567444, -0.41262009739875793, -0.37936997413635254, 3.3844683170318604, -0.04727819934487343, 0.23254601657390594, -0.5442861318588257, 0.9605587124824524, 1.3005222082138062, 0.6727029085159302, -0.5361359119415283, 2.1609458923339844, -0.2739124000072479, -0.12664546072483063, -0.33568477630615234, -0.7400486469268799, -0.6552751660346985, 1.6334185600280762, 0.7532209753990173, -0.3351599872112274, 0.3841952085494995, -0.9331059455871582, 0.29098376631736755, 0.016071751713752747, 2.4093379974365234, -0.4387708008289337, -0.4841739535331726, -1.3081748485565186, 0.39270228147506714, -0.19507189095020294, 0.061372555792331696, -1.8018254041671753, 1.5362253189086914, -0.34923285245895386, 0.9069385528564453, 0.40931040048599243, 0.42970359325408936, 0.660942792892456, 0.6753584146499634, 1.2904350757598877, 0.6973778605461121, -2.2147159576416016, 0.03603136166930199, -1.179612398147583, 0.5501375794410706, 1.9984681606292725, -0.4585285484790802, -1.6222989559173584, -0.622413694858551, -2.0964701175689697, 0.8194953203201294, -0.6315962076187134, -1.3845382928848267, 0.4468449354171753, 0.09535059332847595, 0.3875683844089508, 0.5680106282234192, 0.5095880031585693, 1.7758762836456299, -0.3950616419315338, 1.2547121047973633, -0.33301252126693726, 0.9645686149597168, -0.8052039742469788, -0.32508739829063416, -2.3609931468963623, -0.28935110569000244, -0.35256075859069824, -0.13472773134708405, -0.9918180704116821, -0.43689319491386414, -0.47242841124534607, -1.040303111076355, -1.1281064748764038, -0.6255194544792175, 0.895254909992218, 1.6666253805160522, -0.0795164406299591, -2.5551235675811768, -0.026164861395955086, 0.601041853427887, -1.3783951997756958, 0.1744520664215088, 0.19216963648796082, 1.2862948179244995, -0.7525246739387512, 1.6358627080917358, 0.4422389268875122, -0.6266348958015442, -0.6869161128997803, 0.12049148976802826, 0.9294725060462952, -0.10347910225391388, -0.7835367321968079, -0.9964900016784668, -1.2607636451721191, 0.887509286403656, 0.9954971671104431, -0.9502460956573486, 0.011576410382986069, -2.2391223907470703, 2.5003085136413574, 1.1774200201034546, 0.2238454669713974, 1.019740104675293, -0.19390814006328583, 0.5418007969856262, -0.055931609123945236, -0.5964322686195374, -1.3390192985534668, 1.4117034673690796, 0.47758424282073975, -0.3889204263687134, 0.6106300354003906, -0.4562484323978424, -0.38643428683280945, -0.5748535990715027, 0.4056434631347656, -0.5990669131278992, -0.9185624122619629, -1.8013907670974731, -0.9520278573036194, 0.32421791553497314, -0.40875062346458435, 1.4599610567092896, 1.1979167461395264, -0.8049061298370361, -1.8020983934402466, 1.182795524597168, 0.7273520827293396, 0.012033252976834774, -1.1566340923309326, 0.2548290193080902, 0.4152505397796631, -0.17475122213363647, -0.03973911330103874, -0.10920204222202301, -1.4839515686035156, 0.9470715522766113, 1.1201486587524414, 0.2444034069776535, -0.6712763905525208, -1.3282897472381592, 3.192432165145874, -0.5392500162124634, -0.3495247960090637, -0.6480036973953247, -0.1075221598148346, 2.622624158859253, -0.32713890075683594, 0.46045994758605957, -0.3198500871658325, -0.622113823890686, -0.546775221824646, -0.14415860176086426, 1.2115232944488525, 0.03903697431087494, 0.19101770222187042, 0.33918875455856323, 0.42132681608200073, 0.2732911705970764, 0.5550609827041626, 1.0864765644073486, 0.36792999505996704, -0.7352631688117981, -0.5909485816955566, -1.913561463356018, 1.1378284692764282, 0.4762137234210968, -0.3792227506637573, -1.3636873960494995, -0.06228348985314369, -0.5244929194450378, 0.12489921599626541, -0.6314524412155151, -0.2974832355976105, -0.08534348756074905, -0.979587197303772, -0.581879734992981, -0.2733287215232849, 0.546602189540863, 0.6315075755119324, -1.2371339797973633, -0.7291518449783325, -1.5682286024093628, -2.639827251434326, -0.9921295642852783, 0.6288851499557495, 1.9062002897262573, 1.9182089567184448, 0.3737029433250427, 1.0577747821807861, 1.024393916130066, 0.31028661131858826, -0.29837971925735474, 1.093001365661621, -0.3551842272281647, 0.2071915864944458, 0.05026599392294884, -0.22419989109039307, -0.5026038885116577, 0.1772446483373642, -0.390140563249588, -0.1809779703617096, 1.1866328716278076, 2.258291244506836, 2.084818124771118, -0.18759846687316895, -1.554081916809082, -1.010676383972168, -0.7212377786636353, 0.5685429573059082, 1.3371546268463135, -0.5673259496688843, -0.4228360950946808, -0.33073440194129944, -0.4391044080257416, 0.11206324398517609, -2.2055504322052, 0.05033137649297714, -2.3813161849975586, -3.5429675579071045, 0.5098833441734314, 1.6506197452545166, 1.7737298011779785, -0.9215187430381775, -0.30532804131507874, 1.4006541967391968, 1.192572832107544, 0.35457679629325867, 1.5741479396820068, 1.336687445640564, -0.14985468983650208, 0.2708500325679779, -0.005228765774518251, 0.3567679226398468, -0.23374386131763458, -0.08167829364538193, 0.08645781874656677, 1.1564966440200806, 0.1905835121870041, 1.6894603967666626, -0.3209328055381775, -0.529151439666748, -0.8376772999763489, 0.5622466802597046, 1.02858567237854, 2.4938387870788574, -0.03523929789662361, 1.1997123956680298, 2.3267345428466797, 0.9619269371032715, -0.07346519827842712, -0.39196300506591797, -2.3698954582214355, 0.4960106611251831, 0.3935980796813965, -0.08900552988052368, 0.5945647954940796, 0.4221624433994293, 0.23348668217658997, -0.011260943487286568, -1.120560884475708, 0.07257089763879776, 0.5039306282997131, -0.38201457262039185, 0.2358212023973465, 0.22947779297828674, 1.6445151567459106, -0.545268177986145, 2.5012242794036865, 0.5262479186058044, 1.300431251525879, -0.39257073402404785, -2.154600143432617, 1.33756685256958, -0.6842649579048157, 0.008463931269943714, -1.498746395111084, 0.47545313835144043, 1.9191614389419556, 0.28416141867637634, -1.4522253274917603, -1.116978406906128, -0.021894706413149834, 0.9942949414253235, -0.36599424481391907, 1.3564225435256958, -1.527300238609314, 0.3591661751270294, -1.2054948806762695, -0.691754937171936, -0.34775400161743164, 0.4914230704307556, -1.1264828443527222, 1.4063698053359985, -1.099577784538269, -0.23702624440193176, -0.18977361917495728, 1.356138825416565, 0.6011810898780823, 0.9531124234199524, 0.3494047522544861, -0.5745034217834473, -0.7130845785140991, -0.838853657245636, 1.127086877822876, -1.233534574508667, -0.8642869591712952, -0.3226979672908783, -0.4645378589630127, -0.3750862181186676, 1.2429739236831665, -0.6500008702278137, -0.41570910811424255, -0.7678437829017639, -0.5812945365905762, 1.3052688837051392, 0.2139451950788498, -1.0491095781326294, 0.8559850454330444, 1.5544297695159912, 0.3976323902606964, -0.2642357349395752, -0.7426775693893433, 0.9963453412055969, -1.4302819967269897, 0.7706234455108643, 1.704546332359314, -0.07270032167434692, 1.1278681755065918, -0.18128594756126404, 0.7679879665374756, -0.08983474969863892, -1.2742445468902588, -0.031408537179231644, 0.42140448093414307, 0.7220390439033508, 0.6712164282798767, -0.09799414873123169, 0.1767456978559494, 0.8972131013870239, -0.8681837320327759, 0.08687976747751236, 1.6838151216506958, 0.04049953073263168, 0.6089691519737244, -1.9031401872634888, 2.003865957260132, -0.8130072355270386, -0.4460652470588684, 1.760499358177185, -0.8937281370162964, -0.30819350481033325, -2.1570277214050293, -1.1853773593902588, -0.10120408236980438, 0.04907361418008804, 0.2929381728172302, -0.7143484950065613, -1.839002013206482, 1.5416916608810425, -0.4096807539463043, -0.7374477386474609, 0.5746737718582153, -0.8173245787620544, -1.0269825458526611, 1.845281958580017, 0.6782819628715515, -0.3018801212310791, -0.6630101799964905, -0.11614498496055603, 0.3345932960510254, 1.1702576875686646, -0.7446957230567932, 0.10483676940202713, -1.4219449758529663, -0.03476995229721069, 0.20071114599704742, -0.14202100038528442, -0.49817201495170593, -0.4962832033634186, -0.4739629030227661, -0.28572964668273926, -1.857115626335144, 1.9394346475601196, 0.00511568458750844, 1.5743829011917114, -0.09239626675844193, -0.7926651835441589, -0.8436810374259949, 1.9090908765792847, -1.6296186447143555, 0.7964950203895569, -1.595828652381897, -0.19486117362976074, 1.1135172843933105, -0.008394564501941204, -0.6965034008026123, -0.7378952503204346, -0.7460854053497314, 0.4926929473876953, -0.7971093058586121, 0.9437264204025269, 0.9655888676643372, -1.7991596460342407, 1.4013442993164062, -1.74420166015625, -0.20490016043186188, 0.24137718975543976, 0.14870771765708923, 1.8088189363479614, 0.17048987746238708, 0.019199848175048828, 1.2235931158065796, -0.1440751701593399, -0.06486578285694122, 0.2960744798183441, -1.4925462007522583, 0.7079954743385315, 0.5394129157066345, -2.155836582183838, 0.21119698882102966, -0.3823504149913788, 0.5242404341697693, -0.6945290565490723, 0.6439391374588013, 0.07151070982217789, 1.8071765899658203, 0.6453135013580322, -0.3406580686569214, -0.8524554371833801, -1.8805261850357056, 0.11566787958145142, 1.5989737510681152, -1.924802541732788, 1.4488462209701538, 1.5064315795898438, -0.9173319935798645, -0.47511541843414307, -0.1440081000328064, 1.4307729005813599, -0.6174241900444031, 0.20547513663768768, -0.27101802825927734, -1.0317671298980713, -0.492048978805542, -0.7799707055091858, 0.2227853685617447, 2.106384754180908, -0.1750839352607727, -1.0478142499923706, -0.43339264392852783, -0.9566906690597534, -1.4998506307601929, -0.12896767258644104, 1.0352646112442017, -0.6991934180259705, -0.3713527023792267, -0.10860953480005264, 0.17385107278823853, -0.9340203404426575, 0.9371576905250549, 0.14612433314323425, 1.1122575998306274, -1.9159784317016602, 0.1659604012966156, 0.9456046223640442, -0.00783211924135685, -1.56912362575531, 1.7291146516799927, -0.06700707972049713, 1.3952580690383911, -2.0508193969726562, 0.9374798536300659, -1.2146258354187012, -0.5775865316390991, -0.03056166134774685, 0.2464565634727478, 0.21057668328285217, -0.03622313588857651, 0.6236785650253296, -0.07422031462192535, 1.1866196393966675, -0.45603278279304504, 1.6123982667922974, -0.26193344593048096, -0.02246084064245224, 0.7933541536331177, -0.614436149597168, -0.20357467234134674, 0.9677976965904236, -1.4253708124160767, -0.361169695854187, -0.2307543307542801, -1.1645551919937134, -0.15940557420253754, 1.7641881704330444, -1.0947113037109375, -0.14803080260753632, -0.874274492263794, -2.3897125720977783, 0.8196143507957458, -0.8602834343910217, -0.7885820865631104, -0.18213917315006256, -0.5754677057266235, -0.007196344435214996, -0.2545405924320221, 0.13707438111305237, -0.6133846044540405, 0.6056404113769531, 0.2617022395133972, -0.49604660272598267, -0.6500841975212097, -0.6950111985206604, -0.12208587676286697, 1.2519975900650024, 0.6028143167495728, 0.851337194442749, -0.11651705950498581, 0.33096426725387573, 1.453383445739746, -1.3872302770614624, -0.3329983353614807, 1.0313091278076172, -0.5437090396881104, -2.05950665473938, 2.349867820739746, 0.25798219442367554, -1.469677209854126, 1.0964363813400269, -0.017848515883088112, 1.7155243158340454, 0.8224004507064819, 1.7763121128082275, -1.3731271028518677, -0.6095489263534546, -2.106318712234497, -0.9205277562141418, -0.9338206052780151, -0.2954706847667694, -0.5586636662483215, 1.8314611911773682, -0.21212846040725708, -2.394015073776245, -0.4111219346523285, 0.9101552963256836, -0.8629969358444214, -0.08548121899366379, -0.5782530307769775, 0.30402544140815735, 0.7049210071563721, -0.8063345551490784, 0.33550742268562317, 0.4934980869293213, -1.6836026906967163, 0.5531863570213318, -0.8320940732955933, -0.48952868580818176, 0.5290991067886353, -0.0333014540374279, 1.0293927192687988, 0.6947815418243408, -2.2108235359191895, 0.272339791059494, -1.7585190534591675, 0.4660826027393341, -0.5149959921836853, 0.6004643440246582, 0.30938220024108887, -0.1668969839811325, -2.379852771759033, -0.934755802154541, -0.14517533779144287, 1.0117796659469604, -0.4555515646934509, -0.504378080368042, -1.4339739084243774, 2.1576459407806396, -0.2929539680480957, -0.5601922273635864, -0.4764561653137207, -0.10892939567565918, -0.9054820537567139, 0.34343111515045166, -2.679811477661133, -1.1872543096542358, 0.20690880715847015, 1.0696624517440796, 0.5078275203704834, -0.09926766902208328, 1.600055456161499, -0.20570023357868195, -0.20481842756271362, -0.26238125562667847, -0.419754296541214, 0.04709396883845329, -0.8050892949104309, 1.0817383527755737, -1.1671409606933594, -1.297170877456665, 1.3912088871002197, 0.25007858872413635, 0.7883578538894653, -0.990673303604126, 1.246144413948059, -1.3939985036849976, 0.7515562772750854, 0.9735942482948303, -0.1892968863248825, -1.9053304195404053, -0.18174073100090027, 1.4591599702835083, -0.020416859537363052, -1.0898692607879639, 1.1625113487243652, 2.4017951488494873, 1.6441901922225952, 0.5885197520256042, -0.09249625355005264, -0.11928823590278625, 0.8446840047836304, 0.8051104545593262, 0.4376782476902008, -0.42220234870910645, 0.6280560493469238, 0.4606426954269409, 0.8519623279571533, -0.9667774438858032, 2.4725000858306885, 1.7189282178878784, -1.474492073059082, -0.3568890690803528, 0.24704964458942413, 0.03379594534635544, -1.0598353147506714, 0.4696548879146576, 0.6294484734535217, -0.06258724629878998, -0.9508728384971619, 0.07408116012811661, 0.8823204636573792, 1.9756748676300049, 0.24186304211616516, 0.8613426089286804, 0.7670099139213562, 1.001837968826294, -1.2338956594467163, 1.7421764135360718, 0.6976189613342285, -1.1226422786712646, 0.27291956543922424, -0.5575341582298279, -0.2906922996044159, 0.6675100922584534, -1.1138683557510376, -0.007336026057600975, 0.15236559510231018, -0.48446527123451233, -0.48722973465919495, 0.790945827960968, -0.679418683052063, 0.8629192113876343, -0.08342543244361877, 0.9864001870155334, 0.7462831139564514, 0.5810195207595825, -0.19313795864582062, 1.2244408130645752, 2.4011125564575195, -1.2702523469924927, -1.32418692111969, 0.043983399868011475, 2.5173091888427734, -1.8272812366485596, -0.5172780156135559, -0.9411382079124451, 0.8316216468811035, -0.17097976803779602, 0.10073570162057877, -0.5711592435836792, -1.2858612537384033, 0.31414636969566345, -0.4680486023426056, 1.336600422859192, -0.7687689065933228, 0.6147221922874451, -0.16935226321220398, 0.14809001982212067, -0.10860789567232132, 1.0769340991973877, -1.6823523044586182, -0.5788229703903198, 0.8050779104232788, -1.0657830238342285, 0.1178336814045906, 0.6880485415458679, 1.0754780769348145, -0.1387188583612442, 0.15714557468891144, -0.41369470953941345, 0.5552850365638733, -0.6433817744255066, -0.08671879023313522, -0.2120666801929474, -0.09002033621072769, 1.7304271459579468, -0.6005355715751648, -0.11141137033700943, 1.4105685949325562, -0.6935920715332031, 0.08270400017499924, -0.0857856273651123, -1.2546610832214355, -0.46917274594306946, -1.7624337673187256, 0.5339840054512024, 0.6288599371910095, 0.21194815635681152, -0.8013015389442444, -0.29011455178260803, 0.5883454084396362, 1.5655858516693115, 0.40832263231277466, -2.020433187484741, -0.3188422918319702, 1.1343690156936646, -0.8923438787460327, 0.006932623218744993, 1.0116032361984253, 0.7404350638389587, -1.698319435119629, -0.689485490322113, 1.203761100769043, -0.2578622102737427, -0.6563969254493713, -0.2915245592594147, 0.24298593401908875, 0.3128099739551544, 1.051524043083191, 0.37820640206336975, -0.6528393030166626, 0.6684989929199219, 0.6036516427993774, 0.2879334092140198, 0.6441166400909424, -0.508413553237915, -0.912264347076416, -0.2651650905609131, 1.317600131034851, 1.1075704097747803, 1.1395986080169678, 1.1421340703964233, -0.6621925234794617, 0.510866641998291, -0.44834133982658386, -0.39990147948265076, -0.4432697594165802, 1.4361281394958496, -0.201649010181427, 1.042862892150879, -0.5600792169570923, -2.7840306758880615, -0.1932140588760376, -2.5527307987213135, 0.33472198247909546, -0.1668413132429123, -0.5693341493606567, 0.15144960582256317, -0.9127938747406006, -2.4038259983062744, -0.3789249658584595, -1.7637548446655273, -0.6227508783340454, -0.18206770718097687, -0.2809329032897949, -0.7952303290367126, 0.6290260553359985, 0.6906354427337646, -0.5424806475639343, 0.5121368169784546, 0.37611812353134155, -1.0136951208114624, -0.5207625031471252, -0.40320825576782227, -0.8802111744880676, -0.019711386412382126, 0.43310534954071045, 0.4528869390487671, -0.2768307328224182, 0.3850143849849701, -2.182095527648926, 0.043105531483888626, 0.5284362435340881, 0.0782979428768158, -0.06815361231565475, -0.4106152355670929, -0.5546054840087891, -0.17821529507637024, -0.4238491356372833, -2.0083730220794678, -1.768474817276001, -0.5471705198287964, -0.6218112111091614, 0.38724493980407715, 0.4772305488586426, 0.16943879425525665, 0.7475669980049133, 0.889835774898529, 1.9359766244888306, 0.6142483353614807, -1.3484138250350952, -1.2357081174850464, 2.5169284343719482, 0.25539830327033997, 0.7937415242195129, -0.4008444547653198, 0.4045107364654541, -1.4360703229904175, -1.601118564605713, 0.34475743770599365, 1.2330012321472168, -0.23561686277389526, 0.32682961225509644, -0.3881165087223053, 1.2328521013259888, -0.10824783891439438, -1.3588223457336426, 2.2728195190429688, -0.9097006916999817, 1.3241336345672607, 0.042633213102817535, 0.1822144240140915, 0.4353739619255066, -0.14502887427806854, -0.6753343939781189, -0.5628121495246887, -0.022623280063271523, -0.7760487198829651, 1.177900791168213, -1.42164945602417, 0.6728270649909973, -0.6471058130264282, 0.5255551338195801, -0.517952561378479, 0.8071964979171753, -1.1356168985366821, 1.5000650882720947, -0.2728368043899536, -1.0578091144561768, 0.6304627060890198, -1.1237024068832397, -0.5460718274116516, 1.3200799226760864, 0.6684293150901794, -0.2990837097167969, -0.699491024017334, 0.523946225643158, -0.9087852239608765, -1.2663458585739136, -1.6101115942001343, 0.91238933801651, 0.4573710262775421, 1.1058157682418823, 0.05605659261345863, 1.3217910528182983, -1.2328081130981445, -0.5053418278694153, 1.1566417217254639, 1.1285005807876587, 0.28367599844932556, 0.5985116958618164, -0.212117537856102, 0.0437912717461586, 1.0643181800842285, -1.7637207508087158, -0.3966217637062073, -0.0628429725766182, 0.33314183354377747, -0.9063357710838318, 2.0369913578033447, -1.1061519384384155, 0.06519834697246552, 0.6842378377914429, 0.7526978254318237, 0.9526932835578918, -0.6658565402030945, 0.06648173183202744, 0.36821192502975464, -0.06835149973630905, -0.06076599285006523, -0.3804909884929657, -0.21753908693790436, -0.7286021709442139, 1.0072438716888428, -0.9548628926277161, -0.2399837076663971, 0.3776617646217346, -0.4989333152770996, -0.24854716658592224, 1.6066550016403198, 0.20647092163562775, 1.1743139028549194, 0.1475559026002884, 2.2796247005462646, 0.5564283728599548, 0.5056384205818176, 0.2245904505252838, 0.6813902258872986, -0.18801477551460266, -1.8589617013931274, -1.0215167999267578, -2.8348183631896973, -2.0644733905792236, 2.0344114303588867, 1.1598665714263916, 2.5011487007141113, -0.7682463526725769, -1.1100612878799438, 0.4468945264816284, 0.27833592891693115, 0.5219970345497131, -0.9131054282188416, -1.536738634109497, 1.0190349817276, 0.8302682638168335, 2.4334323406219482, 0.116155244410038, 3.296583890914917, 0.04085089638829231, 0.14003099501132965, 0.3295910656452179, 0.6420074105262756, -1.5967093706130981, -0.885112464427948, -0.6535990834236145, 2.4378762245178223, -1.0908697843551636, -0.6392540335655212, 0.8886602520942688, 0.942479133605957, -1.0680162906646729, 1.9644780158996582, 0.11881304532289505, 1.105741024017334, 0.991860032081604, 0.5347906947135925, -0.5409600138664246, -1.5906994342803955, -0.8863663673400879, 0.3145420253276825, -1.184556007385254, 0.5981714725494385, -0.9303840398788452, 0.7018967270851135, -0.2000168263912201, -0.5177063345909119, -0.005377970170229673, -0.1496669352054596, 0.4575062692165375, -0.21806295216083527, -1.5416975021362305, 0.7798487544059753, -0.3161563575267792, -2.9996724128723145, 1.1771154403686523, 1.1720691919326782, -1.42130708694458, -0.6114386320114136, -0.018857182934880257, 1.4498953819274902, 0.8993534445762634, -0.9685867428779602, -0.28276732563972473, 0.7314795255661011, -1.6019634008407593, 1.385502576828003, -0.5689979791641235, 0.2854747772216797, 1.2966066598892212, -0.8381891250610352, 0.17500261962413788, 0.006027581170201302, 0.8720093965530396, -1.0846573114395142, 0.1683686077594757, 0.32276177406311035, -0.38410481810569763, 0.5920903086662292, 1.5063270330429077, 0.6080256104469299, 0.7853367924690247, -1.582865595817566, -0.7175677418708801], [-1.3542646169662476, 1.1495567560195923, 0.3745271563529968, 0.03599359840154648, 0.6191831827163696, -1.2620059251785278, 0.4026302695274353, -0.527283787727356, -1.026094675064087, 0.6835550665855408, 0.5356741547584534, -1.8444877862930298, -0.23532740771770477, 1.5992300510406494, 1.2757439613342285, 0.21113061904907227, 0.21292538940906525, 1.1811269521713257, -1.621106743812561, 0.6169980764389038, -0.2825905978679657, 0.0328470803797245, -1.2628110647201538, 1.7319976091384888, 0.952045202255249, -1.4987531900405884, -0.4840964078903198, -0.196431502699852, -0.7327600121498108, -0.5087536573410034, 0.3281940221786499, -0.41925248503685, -0.4190903306007385, -0.9821982979774475, 0.7112870812416077, -0.04864524304866791, 0.36918187141418457, -0.41753289103507996, -1.664416790008545, -1.1104389429092407, 0.9294427037239075, 0.09185943752527237, 0.9925876259803772, -1.2020431756973267, -0.4934542775154114, -0.6199666261672974, -0.9893078207969666, -0.9172034859657288, -0.32199716567993164, -0.3881458342075348, -0.47009795904159546, -0.36108601093292236, -1.3968324661254883, 1.4736343622207642, -0.0741904228925705, -0.43372154235839844, -0.6972594857215881, 1.1095844507217407, 0.03319726511836052, -1.0517191886901855, -2.4062070846557617, 1.9903373718261719, 1.1485300064086914, -0.14587071537971497, -0.4262557923793793, -0.07704517990350723, -0.028758326545357704, -1.3511680364608765, -0.2957921326160431, -1.987646222114563, -0.298019140958786, 0.11449426412582397, 1.1766167879104614, -1.5876508951187134, 1.323006272315979, -0.3718750774860382, -0.2654281258583069, -0.09851600229740143, -0.7546308636665344, -0.08641711622476578, 0.2749064564704895, -0.3840329945087433, -1.658310890197754, -0.8580965995788574, 1.0519920587539673, -1.467100977897644, 0.3593783974647522, 1.594077229499817, -0.08761713653802872, 0.19688762724399567, -0.342146098613739, 0.6674664616584778, 0.3407857418060303, -0.366608589887619, -0.29141268134117126, 2.143644332885742, 1.6157501935958862, -0.3560062348842621, -0.9199876189231873, 0.7677261829376221, 1.5042898654937744, -0.08333103358745575, -1.5493336915969849, -0.1944812685251236, 0.11702635139226913, 0.45875877141952515, 0.5397542119026184, -2.1401402950286865, 0.6504780054092407, 0.7611957788467407, 0.8778930902481079, -0.65944904088974, -0.003328672144562006, 1.0077170133590698, 1.1040465831756592, 2.0367271900177, -0.475725919008255, -1.8536057472229004, 0.49536243081092834, 0.6998342275619507, -0.6383841037750244, 2.352841854095459, 0.7070732712745667, 1.918355941772461, 0.28254786133766174, 0.181104376912117, -1.066089153289795, 1.0335346460342407, 0.7301533222198486, -2.228980541229248, -1.1030865907669067, -2.1233315467834473, 0.05767300724983215, -1.1426228284835815, 1.4985899925231934, -0.18096663057804108, -0.5074804425239563, 2.3407552242279053, -1.5281121730804443, -0.09656616300344467, 0.3691624701023102, -0.3930843472480774, 0.18619076907634735, 0.8205493688583374, -0.6782922744750977, -0.45622631907463074, -0.7214857339859009, 0.014197789132595062, 1.049493432044983, 0.6707140207290649, 1.3234734535217285, 1.0261937379837036, 1.2632052898406982, -0.945482075214386, -0.10428670793771744, 0.34891393780708313, -0.3430003523826599, -1.2466931343078613, 1.6020289659500122, 0.5775207281112671, 0.8452780842781067, -0.08740496635437012, -0.9747108817100525, -0.30536237359046936, 1.929614543914795, 0.6016175150871277, 2.298915386199951, -0.1934422254562378, 0.41781482100486755, -0.08016592264175415, -0.935230016708374, 1.150066614151001, -0.7145572900772095, -1.8518192768096924, -2.3199501037597656, 2.846038341522217, 2.47430419921875, 2.144737720489502, 0.7553079724311829, 0.48922958970069885, 1.0311627388000488, 0.7393538355827332, 1.3591899871826172, -1.1574101448059082, -0.8072368502616882, -0.41575512290000916, -1.7197027206420898, -1.1508558988571167, 1.841310977935791, 1.8541069030761719, 0.29674065113067627, -1.5691885948181152, -0.4027461111545563, -1.5721129179000854, 1.1750625371932983, -2.4129810333251953, -2.0849103927612305, 1.7131956815719604, 0.15231648087501526, -0.4651757776737213, 1.706803321838379, -0.8817654848098755, 0.1421533077955246, 1.2613776922225952, 1.1758129596710205, 0.13567321002483368, 1.4719443321228027, 1.7200630903244019, 1.6434890031814575, 0.1649996042251587, -1.906849980354309, 0.7457695603370667, -0.535515308380127, 1.231110692024231, -1.6356066465377808, -0.7918795943260193, -1.042510986328125, -1.44212806224823, -1.7224966287612915, -0.13305003941059113, -0.30615219473838806, -1.3683863878250122, -1.845118761062622, 0.37490352988243103, -0.24637296795845032, -0.189091756939888, 0.4671521484851837, -0.5512995719909668, -1.0935777425765991, -0.722172737121582, 1.274505853652954, 1.0718426704406738, -1.4877359867095947, 0.38264453411102295, -1.8261682987213135, -0.04167108237743378, -2.153646945953369, -0.2049959897994995, -1.3005794286727905, -1.1254481077194214, 0.04774310812354088, -1.2163439989089966, 0.9300553798675537, -1.5894544124603271, -0.4297384023666382, -3.31866192817688, -0.5428176522254944, 1.0267412662506104, 0.5845681428909302, -0.021868843585252762, -2.073150396347046, 0.18572546541690826, 0.11695560812950134, 0.18210679292678833, 0.39689961075782776, -0.9673024415969849, 0.4656801223754883, 0.9441999197006226, -0.27983149886131287, -0.45341140031814575, -1.5953530073165894, 1.0715177059173584, -0.12060504406690598, 0.39353683590888977, -0.06882255524396896, 1.0891073942184448, 0.3741447925567627, -0.7695006728172302, 0.3206258714199066, 1.2124370336532593, -1.0211156606674194, 0.07376037538051605, 0.5109211802482605, -0.22015637159347534, -1.9678125381469727, -0.31839001178741455, -0.3794640600681305, -2.0860862731933594, -0.1072680801153183, 1.2370827198028564, -0.018792303279042244, -1.2330862283706665, 1.2479811906814575, -0.09932219982147217, 0.23706082999706268, 0.5839447975158691, -1.368751883506775, -1.2576172351837158, 0.27972733974456787, 0.17707638442516327, -0.057150375097990036, 1.1795189380645752, 0.08150725811719894, 0.3324219584465027, -1.4584101438522339, 0.27845773100852966, 0.647488534450531, -0.7814022302627563, 1.8085064888000488, -1.2725934982299805, 1.0041025876998901, 0.24639791250228882, -1.0585336685180664, 0.7890924215316772, 0.7194098234176636, 0.5231953859329224, 1.3285479545593262, 0.2778262794017792, 0.4277343451976776, 0.5301883816719055, 0.6954048275947571, 1.122246503829956, 0.5208264589309692, -0.7264543771743774, 0.09232562780380249, -0.8818853497505188, 0.8273704051971436, -0.9278144240379333, 0.34638160467147827, -1.4764500856399536, 0.758579671382904, 0.14331857860088348, -0.49674686789512634, -1.508273959159851, -0.40597179532051086, 0.2410571426153183, 0.9164498448371887, -0.08942605555057526, 0.6071053147315979, -0.272225558757782, -0.03998695686459541, -1.4240984916687012, 1.2999202013015747, 0.8563091158866882, -0.1813359260559082, 0.9492132067680359, -0.2406189739704132, 1.438356637954712, -0.7024730443954468, -1.7546781301498413, 0.7500275373458862, 0.1041160374879837, 0.4227138161659241, 0.6476534008979797, -0.878829836845398, 0.7897257804870605, 1.3265550136566162, 0.23966409265995026, 0.3275817632675171, 1.014807105064392, -1.1565945148468018, 1.2346611022949219, -1.0933159589767456, 1.6302120685577393, -0.17337459325790405, -0.2848508059978485, -0.1719411164522171, -0.056986674666404724, 1.717917561531067, 0.1160164624452591, -0.7337120771408081, 1.9104218482971191, 0.011295527219772339, -0.199677973985672, 1.094290852546692, 0.7045323252677917, -0.19077366590499878, 0.21981391310691833, 1.7948367595672607, 0.5166721343994141, -0.31168174743652344, 0.1780727207660675, 0.8508645296096802, -0.012376775965094566, -0.4462644159793854, 1.3086360692977905, -0.7247142791748047, 0.8466644883155823, -2.6653454303741455, -1.0006901025772095, -0.31510624289512634, -0.5575198531150818, 0.3401992917060852, 2.064213752746582, 0.7625582814216614, -0.3253912329673767, 0.09942755103111267, -1.3024741411209106, 0.5082759857177734, -1.1463775634765625, -0.1630144715309143, -0.24461308121681213, -1.5851385593414307, 0.31294456124305725, -0.05619794875383377, 2.385061264038086, -1.5681160688400269, 1.8931511640548706, 0.6656240224838257, -1.6675878763198853, -0.33275237679481506, -2.9697630405426025, 1.2764793634414673, -1.9474986791610718, -0.37121105194091797, 0.4916924834251404, 0.8561903834342957, -0.4538915157318115, 0.4492330253124237, -0.9812260270118713, 1.291412353515625, 0.6067607402801514, -1.3116984367370605, -0.7351610660552979, -0.284533828496933, 2.0309839248657227, 0.9643575549125671, -1.8399949073791504, 1.837459683418274, -0.732742190361023, 0.3860560953617096, -0.15426769852638245, 0.4534689784049988, -0.28595614433288574, -0.8778231143951416, -0.006109577603638172, 0.03348139300942421, 1.2406398057937622, -0.9813918471336365, -1.0527762174606323, 1.1841530799865723, -0.1349211484193802, -2.0920557975769043, -1.061203956604004, -1.7381771802902222, -1.2321058511734009, -1.4646575450897217, 1.761276125907898, -0.8852013349533081, -1.1146186590194702, 2.6289920806884766, -0.49063751101493835, 0.419974148273468, -0.5169654488563538, -0.9442557692527771, 0.6283194422721863, 1.9375513792037964, -0.6144128441810608, -1.0179402828216553, 0.8296866416931152, 1.6951051950454712, -0.3631415367126465, 0.4591749310493469, -0.29004141688346863, -1.2635046243667603, -1.3431103229522705, 0.4486519992351532, 0.7034425735473633, 0.9123783707618713, -0.05411975458264351, -1.333357572555542, -0.11542432755231857, 0.816146731376648, -0.38661760091781616, -0.4496946930885315, 0.834000825881958, -0.9491202235221863, -0.7023788094520569, -1.0025341510772705, 1.0377601385116577, -0.8048997521400452, -0.26946204900741577, 1.1457351446151733, -0.778425395488739, 1.1954505443572998, -0.6128268837928772, 0.11370459944009781, -1.3487194776535034, 0.7749868035316467, -1.55853271484375, -0.4154419004917145, -1.167047142982483, -0.6111412644386292, -0.8320674896240234, -0.480337530374527, -0.3052077293395996, -1.253352403640747, 1.8503278493881226, 0.1309313327074051, -0.8756036758422852, 0.25498202443122864, -0.45500174164772034, -1.537900447845459, -1.3020150661468506, 1.5371111631393433, 0.694943904876709, -0.47583794593811035, 2.3250982761383057, -0.3908231854438782, 0.5628311634063721, 1.0576192140579224, 0.07884260267019272, -1.343044638633728, 0.019969239830970764, 0.4734455347061157, -0.6000180244445801, 0.5139465928077698, -0.5913829803466797, -0.10209953784942627, 1.1275430917739868, -0.9911660552024841, 0.5884270668029785, -0.6454065442085266, 0.9478386044502258, 0.7278352975845337, -0.77677983045578, 2.362229585647583, 0.6374277472496033, 0.48092028498649597, 0.297683447599411, -0.4190966784954071, -0.8967857360839844, 0.6297850608825684, 1.0206648111343384, -0.11123280227184296, 1.382651448249817, 0.23791290819644928, -0.48433545231819153, -1.0472806692123413, -1.3668467998504639, 0.6519150137901306, 0.5540313124656677, -1.7215654850006104, 0.4473518133163452, -0.12331268936395645, -0.9495335817337036, -2.163816213607788, 0.6140503883361816, 0.45586222410202026, 1.1774406433105469, -0.14428037405014038, 0.9348194003105164, 2.7604928016662598, -0.7984036803245544, -0.7818652391433716, 2.1959071159362793, 0.9356593489646912, -0.7877044081687927, -1.2982878684997559, -0.7553954124450684, -1.0601835250854492, 0.47903725504875183, 0.6573916673660278, -1.8584061861038208, 0.4527602195739746, 0.6982101798057556, 0.6773503422737122, 0.31275486946105957, 1.6923208236694336, 0.14804059267044067, 0.15789489448070526, -0.8072845935821533, -1.6362837553024292, 1.0412927865982056, -0.10539046674966812, -0.3820647895336151, 0.1995851993560791, -0.2670189440250397, 0.5713584423065186, -0.09005743265151978, -0.5359631776809692, 0.2071048468351364, 0.37369269132614136, -0.6401896476745605, 1.8969919681549072, 0.25342652201652527, 0.11338088661432266, 0.41908931732177734, -1.5317137241363525, 0.6405224800109863, -0.021210668608546257, -0.5670639872550964, -0.8593438863754272, -0.13155154883861542, -2.009763717651367, 0.6383091807365417, -0.05452568084001541, -0.7905638217926025, 1.1853781938552856, -0.8660032153129578, -1.1514426469802856, -0.3950519859790802, -0.510940432548523, -1.07916259765625, 0.26003411412239075, 1.0983896255493164, -1.369622826576233, -0.9987757802009583, -0.037695303559303284, -0.928203821182251, 0.5687654614448547, 1.1224277019500732, -1.5949245691299438, 0.9421014189720154, -0.7616243958473206, 0.45742735266685486, 1.5198205709457397, -0.8618493676185608, -0.7337480783462524, 0.11037804186344147, -2.397109270095825, -0.07785163819789886, 0.15084350109100342, -0.7562969923019409, -1.5702989101409912, 1.061653971672058, -0.9157474040985107, 0.5490899085998535, 0.8685299158096313, -1.722694993019104, -0.14554716646671295, -0.4665551483631134, 0.30858293175697327, 1.8024929761886597, -0.49789807200431824, 0.42459043860435486, 0.16828685998916626, -0.5908886790275574, 1.1485223770141602, 0.1867581158876419, 0.3462722599506378, 0.17227865755558014, -1.4120701551437378, -0.839078426361084, 0.03868807107210159, 0.5169333219528198, -1.670350432395935, -0.8458612561225891, -0.19895727932453156, -0.3093803822994232, -0.5626093149185181, -0.7902058959007263, -0.5753894448280334, 0.042829059064388275, -0.02773940935730934, -0.6294649243354797, -0.15410351753234863, -1.3870915174484253, 0.30029627680778503, 1.0713344812393188, -0.20922274887561798, -0.5400494337081909, -0.4969332218170166, 0.27515190839767456, -0.8263818621635437, -0.499021053314209, -0.47083979845046997, 0.07497306168079376, -0.5243374109268188, 1.750773549079895, 0.1447359323501587, 0.23474179208278656, -0.9278759360313416, 0.605278491973877, 0.375457227230072, 0.47584712505340576, 0.8866193294525146, -0.3925503194332123, 1.0818092823028564, -0.05804560333490372, 0.07236815243959427, -0.9884893298149109, 0.428257554769516, 0.7341750264167786, 0.4841792583465576, -0.06215016916394234, -0.028649775311350822, 1.1281628608703613, 0.9963968396186829, -0.5090944766998291, -1.029170274734497, 1.0327637195587158, 1.4395607709884644, -0.43852075934410095, -0.9306878447532654, -0.7996318340301514, 0.8226134181022644, -0.42137280106544495, 0.39485353231430054, -2.4669575691223145, 0.7463653683662415, -0.12409964948892593, 1.3669153451919556, 0.40624499320983887, 0.006589402444660664, 0.7110879421234131, 0.1195206567645073, -0.8238638043403625, -0.3845605254173279, 0.28626126050949097, 1.1153188943862915, -0.9247187972068787, -0.29399344325065613, 1.7048697471618652, 0.32468464970588684, -0.7818419933319092, -0.7779103517532349, -1.7214186191558838, -0.20177890360355377, 0.1685919463634491, -2.299051284790039, -1.2453912496566772, -0.1739024817943573, -0.11598765105009079, 0.1548151969909668, -2.9967920780181885, 0.8579975962638855, -1.2464616298675537, -0.49855226278305054, 0.15581871569156647, 1.1388543844223022, 1.3313292264938354, -0.25807201862335205, -0.7262607216835022, -0.42380091547966003, 0.2542918920516968, 0.2344033569097519, 1.7522231340408325, 0.005713304039090872, 1.1399048566818237, 0.009918268769979477, 0.11584564298391342, 1.5636317729949951, -0.4049302041530609, -0.3821682929992676, -1.152870535850525, 0.0675937831401825, 1.2693647146224976, 0.23799115419387817, -1.5189564228057861, -0.8888655304908752, 0.9143125414848328, -1.3566995859146118, 0.21817295253276825, -1.3863744735717773, 1.5542210340499878, 1.1671525239944458, -0.35616979002952576, 0.7663818001747131, -1.144040822982788, 0.3704150915145874, -1.633056402206421, -0.1673170030117035, 0.36408865451812744, 0.1120203509926796, -1.142240285873413, -1.0189927816390991, 0.4709406793117523, -1.0527565479278564, -0.0965019166469574, -0.09140557795763016, 0.6747688055038452, 1.06868577003479, -0.7935543656349182, -0.04123418778181076, 1.2700631618499756, -0.08953435719013214, 1.2833468914031982, -0.8221092820167542, -1.4305740594863892, -0.5539798736572266, 0.32257533073425293, -0.022160528227686882, -0.6301864385604858, -1.8576724529266357, 0.3340699076652527, -0.639860212802887, -0.04294991120696068, -0.5864004492759705, 0.1600489467382431, -2.219802141189575, -1.699904203414917, -0.565126359462738, -1.094895362854004, 0.14202599227428436, 1.1512762308120728, -0.48379454016685486, 0.5357025861740112, 1.097241759300232, 0.4727626144886017, 0.22292709350585938, 1.5896059274673462, 0.16931575536727905, 2.0047061443328857, -0.6567842364311218, -1.235592246055603, 0.44807103276252747, -1.2022422552108765, -0.7937237620353699, 0.03620556369423866, 0.18992897868156433, -0.2672426104545593, -0.11100775003433228, 0.1405043751001358, 0.47785496711730957, 0.5370975136756897, -0.4650322496891022, 0.05897559970617294, -0.7235539555549622, 0.68526691198349, -1.0691277980804443, 0.4324953556060791, -0.0627380758523941, -1.3089455366134644, -0.5718945860862732, -0.9570289254188538, -0.32899364829063416, 0.875920295715332, -0.5589437484741211, -0.5411231517791748, 0.6497879028320312, 1.1681050062179565, 0.5969669222831726, -0.4875859022140503, 0.4788987934589386, -0.4299287796020508, -1.2664819955825806, 0.9415683150291443, 1.2912487983703613, 0.5939888954162598, 0.42518723011016846, -0.2624174654483795, 1.1246728897094727, 0.6684484481811523, 1.0251718759536743, 2.341144323348999, -0.22655509412288666, 0.3817678391933441, -1.0719630718231201, 0.38390472531318665, -0.6674644947052002, -0.16893114149570465, 1.311453104019165, -0.7929648160934448, -0.6958339214324951, -0.46568456292152405, -1.2517460584640503, -1.27717125415802, -1.3701927661895752, 1.3642830848693848, -1.0100924968719482, -0.45644253492355347, -0.02263643406331539, 0.39339402318000793, 0.8018644452095032, -0.6908888220787048, -1.2194432020187378, -0.39568567276000977, 0.845304548740387, -0.018860939890146255, 0.8548625111579895, -0.9734957218170166, -0.8929722905158997, 1.3862168788909912, -0.15174219012260437, 0.4078148305416107, 0.13085125386714935, -0.5798578858375549, -0.10315442085266113, -0.8918983340263367, -1.7591888904571533, 0.12373191863298416, 1.932847023010254, 1.152708649635315, 1.741357684135437, -0.8313584327697754, -1.8616597652435303, -0.24237248301506042, 0.8344513177871704, 0.04479898512363434, -1.5920791625976562, -0.3854620158672333, -0.8553760647773743, 0.4414701461791992, -0.6386474967002869, 0.25379717350006104, 0.8181407451629639, -0.03251338750123978, -1.1060199737548828, -0.7068474292755127, 0.06944452226161957, 0.07622247189283371, -1.6537500619888306, -0.4274130165576935, 0.8562620878219604, -1.0109565258026123, -0.3658285140991211, 1.2741838693618774, 1.1216728687286377, 0.3507866859436035, 1.0892198085784912, 1.8637112379074097, -1.532790184020996, -1.2994399070739746, -1.8560892343521118, -0.668485164642334, -1.1751433610916138, 0.08281990885734558, -1.9162863492965698, -0.608836829662323, -0.9929649829864502, -1.2117431163787842, -0.24460576474666595, 0.979509711265564, 0.5811188817024231, 0.27953267097473145, 1.097720980644226, -0.036493152379989624, 0.8669057488441467, -1.2854284048080444, -0.4269864559173584, 0.021518591791391373, -0.13033731281757355, -0.14107897877693176, -0.5136457085609436, 0.013372328132390976, -0.6026788353919983, -0.11207947880029678, 0.07180707156658173, -0.33738335967063904, 0.2780008614063263, -1.9354324340820312, -0.3215479850769043, -0.1744881570339203, -1.6148300170898438, -0.43283775448799133, 2.1487977504730225, 0.23135894536972046, -0.05572584643959999, 0.06974141299724579, 1.13506019115448, -0.8590519428253174, 0.5592306852340698, 0.7376121282577515, 0.7269995808601379, -0.5098750591278076, 1.0249706506729126, 0.965287446975708, -2.6137256622314453, -0.014858915470540524, -0.22996832430362701, 0.7362406849861145, 0.6185839176177979, -1.360425353050232, -0.0999433621764183, -0.9970879554748535, -0.5900167226791382, -0.5924866795539856, 0.20837675034999847, -1.2420120239257812, 0.3980320692062378, -1.1078174114227295, -0.6900467276573181, -1.2513986825942993, 0.3869415819644928, -1.2487812042236328, 0.7676643133163452, -0.7035082578659058, -0.6674354672431946, 1.297192931175232, -1.7618216276168823, 0.5524778962135315, -0.5349332690238953, 0.21677926182746887, 0.8255569338798523, -0.5247205495834351, -0.16517877578735352, -1.373883843421936, 0.397774875164032, 0.23881791532039642, 0.5839869379997253, 0.3442881405353546, 0.5964995622634888, -1.450986385345459, 1.272254228591919, 1.1521461009979248, -1.3647252321243286, -0.6701651215553284, -1.2062016725540161, 0.2445482313632965, 0.321230411529541, -1.9317991733551025, 0.6712756752967834, -0.17382927238941193, -0.22331330180168152, -1.153573989868164, -1.0190579891204834, -0.1792765110731125, 0.9607180953025818, -0.07020413130521774], [1.307187795639038, -0.8685872554779053, -0.5753854513168335, -1.609528660774231, 1.0080933570861816, -1.6022512912750244, 0.03820056468248367, 0.36820071935653687, -0.49221065640449524, 0.3276534080505371, -0.38920363783836365, -0.329581081867218, -0.4508781135082245, -0.45299211144447327, 0.6590209007263184, 0.3803202211856842, 0.20435695350170135, -0.2577323615550995, 0.1496373862028122, -0.06361223012208939, -0.8096859455108643, 1.1646249294281006, 0.3269283175468445, 0.42841771245002747, 0.9001417756080627, 0.32641443610191345, -1.830308198928833, -0.49467751383781433, -1.8678653240203857, -0.33775365352630615, 1.0921822786331177, 0.7090123891830444, -1.2443784475326538, -0.5147417783737183, 0.9310961365699768, -1.0233380794525146, 0.4808003604412079, -0.8171783685684204, -1.401787519454956, 0.8078743815422058, -0.3187079429626465, -0.4712984263896942, 0.5496507287025452, -0.21708977222442627, 0.053550176322460175, -0.6521735787391663, 0.3188130855560303, -0.3922572135925293, 0.1975812017917633, 1.0207500457763672, -0.9698506593704224, -0.8899236917495728, 0.9914593696594238, -0.884381115436554, 0.4708156883716583, -0.0712578296661377, -0.09241536259651184, 0.7797442674636841, 0.9750664830207825, 0.3017346262931824, 0.7010027170181274, -0.8835291862487793, -0.3151910901069641, -0.8221455812454224, -1.038490891456604, -0.6038610339164734, -1.093428134918213, -1.6798175573349, -0.13903498649597168, 0.12929752469062805, 1.634770393371582, 2.455183506011963, 1.3633126020431519, -1.2903530597686768, -0.5800521373748779, 0.6181479692459106, 1.2227787971496582, -1.4895954132080078, -1.1120442152023315, -0.7288857102394104, -0.24124419689178467, 0.9562284350395203, -1.7427161931991577, 1.4941238164901733, -1.5135977268218994, 0.5880029797554016, 2.6004533767700195, 0.6025485396385193, -0.9412236213684082, 0.5810254216194153, 0.5610688328742981, -0.5131603479385376, -1.461044430732727, -0.6704360842704773, -0.09542573243379593, 0.12191596627235413, 0.5961014032363892, -0.5739267468452454, 0.4973251223564148, 0.8283689022064209, -0.8741931319236755, 0.21678972244262695, -0.02883736602962017, 1.3365956544876099, 0.38391682505607605, 0.6554083228111267, 0.1764838546514511, 1.1928634643554688, -0.8768047094345093, -0.7914597392082214, -1.0341547727584839, 0.0414743646979332, -0.6787872910499573, -1.3030438423156738, 0.6601332426071167, 0.6454576253890991, -2.2209510803222656, -1.7704156637191772, -0.13675114512443542, 1.2243057489395142, 0.8880713582038879, 0.09256661683320999, -1.110275387763977, 1.4795215129852295, 0.8998793959617615, -0.4364912807941437, -0.6822623610496521, -1.16097092628479, 0.986305296421051, -0.48466581106185913, 0.9610456824302673, 0.061736635863780975, -0.7031830549240112, 0.5857332348823547, -0.5855982899665833, -1.6612281799316406, -1.5429965257644653, -0.6281660795211792, -1.2945692539215088, -0.830669105052948, 0.1047571450471878, -0.22933721542358398, 1.6027297973632812, -0.5975714325904846, -0.004361962433904409, 0.3962728977203369, -0.18533839285373688, 0.27623000741004944, 0.4166838228702545, 1.9626625776290894, 0.7530367374420166, -0.40445390343666077, 1.193289041519165, -0.7597149014472961, 0.3226364254951477, -1.6019439697265625, -0.4433860778808594, 0.7056628465652466, -0.2699902355670929, 0.027541177347302437, -0.512828528881073, -1.6209216117858887, 1.6270769834518433, -1.3286300897598267, 1.0354430675506592, 0.14734092354774475, -0.5264033079147339, -1.9635816812515259, 0.8490915894508362, 0.17715269327163696, -0.2051781862974167, 0.08175981789827347, 0.8089094758033752, 1.450427770614624, -1.4393497705459595, 0.11866585165262222, -2.201629400253296, -0.8549988269805908, 1.0540684461593628, 0.6068093180656433, 0.2586899399757385, 0.5126243233680725, 0.846413791179657, -0.7999319434165955, -0.7071583271026611, 1.2612510919570923, -0.8369940519332886, 0.09397516399621964, -1.422494888305664, 0.4839118421077728, -1.793463945388794, 0.20669396221637726, 1.753823161125183, -1.1639994382858276, 0.35284221172332764, -0.08218705654144287, -0.1997276395559311, -0.5749725103378296, 1.0222584009170532, 0.2770999073982239, 0.8785882592201233, 0.29082030057907104, 0.48208266496658325, -0.5405625700950623, -0.43295347690582275, -2.6252193450927734, 0.3591744899749756, 0.9073582291603088, -0.27401983737945557, -1.6551364660263062, 1.7557569742202759, 1.9878394603729248, -0.2722161114215851, 0.15771980583667755, 0.018919093534350395, 0.9745885729789734, 0.8800183534622192, 0.28560909628868103, 1.0652880668640137, 0.07693855464458466, -1.0547523498535156, -0.02105814218521118, 0.8958985805511475, -0.7475829720497131, -0.5021466612815857, 0.12750910222530365, -0.47867274284362793, -0.3127177655696869, -0.9397487044334412, -1.3850632905960083, -0.39268022775650024, 0.07733870297670364, -0.0513063482940197, 1.5170745849609375, 0.31005600094795227, 0.31708812713623047, 0.3053382337093353, -0.29828840494155884, -0.5187252163887024, 0.4547334909439087, -0.0472324900329113, 1.9176992177963257, 0.2375006079673767, -1.5615051984786987, -0.37418845295906067, 1.8645480871200562, -1.137711763381958, -0.3690042495727539, -0.35921627283096313, 0.20699059963226318, 0.3336663544178009, -0.8075066804885864, -0.757714569568634, 0.36263614892959595, 1.634285569190979, 0.25425416231155396, -1.526407241821289, -0.6597191691398621, -0.07230282574892044, -0.8125987648963928, -0.07784027606248856, -0.3542684018611908, 0.02437490224838257, 0.5017791986465454, -2.197801113128662, 0.8541887998580933, 1.1723909378051758, 0.7364023327827454, -0.01421385444700718, 0.2598029375076294, 2.1610562801361084, 1.4093194007873535, -0.018822714686393738, 0.8257951140403748, 0.31065675616264343, -1.2925808429718018, -0.7970237135887146, 0.4129348695278168, -1.3076626062393188, 0.6887454986572266, 0.575278103351593, 1.6092519760131836, 0.017384227365255356, -1.6939457654953003, -0.3782947361469269, -0.5909169912338257, -0.5143182873725891, -0.06261099129915237, -1.7867445945739746, 1.0981532335281372, 0.49595123529434204, -0.4646267294883728, -0.1828591376543045, 0.061147790402173996, -0.07329436391592026, -1.0181059837341309, -0.921370267868042, -1.3447450399398804, 0.4849357306957245, -0.7990272641181946, -0.5053874850273132, -1.5548467636108398, -0.03281436488032341, -0.9718114733695984, -0.8821970820426941, -2.134857416152954, 0.9522486925125122, 0.10819532722234726, 0.2360609471797943, -2.0840182304382324, -1.884204387664795, 1.4873034954071045, -0.4974699020385742, -0.1673143059015274, -0.05120202153921127, 0.11194189637899399, -0.00649813748896122, -1.1482971906661987, 0.8178497552871704, -0.6458467841148376, -0.20798125863075256, -1.4420146942138672, -1.7256686687469482, 0.8306668996810913, -1.4130992889404297, -1.721554160118103, -1.0723246335983276, 0.378785640001297, -0.2912886142730713, -0.6760950088500977, -1.060102939605713, -0.33190762996673584, -0.9864497184753418, -0.8168679475784302, -0.5591674447059631, -1.0157755613327026, -0.567794919013977, 0.801035463809967, -0.45461276173591614, 1.6623010635375977, -0.5870884656906128, -2.6277379989624023, -0.08844684064388275, 0.6719110012054443, -0.06473632156848907, -1.892642855644226, -0.3231038749217987, -1.7665256261825562, 0.8580392599105835, 0.5304247736930847, 0.27292314171791077, 1.0685604810714722, -0.8255398273468018, 0.013615889474749565, -0.3631298840045929, -0.9504479765892029, -0.3816438317298889, -1.524263620376587, -1.7544747591018677, -0.9611214995384216, -0.037163496017456055, 0.25953802466392517, -0.08661375194787979, 2.2346205711364746, 0.9776848554611206, -0.47701531648635864, 0.8263320922851562, -1.005403757095337, 0.48044243454933167, -0.5254747867584229, 0.3102215826511383, -1.1315577030181885, 1.3476336002349854, -0.2011861652135849, 0.6842817068099976, 0.38579297065734863, -2.6614348888397217, 0.3738093376159668, 0.30389368534088135, -1.6337145566940308, 1.4113495349884033, 0.5116433501243591, -1.0374218225479126, 0.7313273549079895, -0.443685919046402, -0.42894884943962097, 0.7047721147537231, -0.22293011844158173, -1.1107780933380127, 0.03494589403271675, 0.3720880150794983, 0.7571253776550293, 0.3908412456512451, -0.3699364960193634, 0.70328688621521, -1.519095778465271, -0.44140464067459106, -0.28969183564186096, 0.7011076807975769, 0.16139622032642365, -0.0009976079454645514, 0.6124628186225891, -0.9646650552749634, -0.6641655564308167, 0.23817497491836548, -0.09632843732833862, 0.08275580406188965, -0.618524968624115, -0.7715355753898621, -0.3684723973274231, 1.311316728591919, 0.36316415667533875, 0.3344014883041382, -1.5983766317367554, -0.8099855780601501, -1.1323156356811523, -0.40909427404403687, -1.1415095329284668, 0.5460498332977295, 0.08516452461481094, -0.09639070183038712, 0.7722142338752747, -0.44583842158317566, -1.3297133445739746, 0.14644959568977356, -0.8800665140151978, -1.446388602256775, -1.00503671169281, -0.6300265789031982, -0.6009039282798767, 0.16456544399261475, -0.3272649645805359, -0.14538390934467316, -0.9480281472206116, -1.0480141639709473, -0.07915555685758591, 1.4011805057525635, -0.7422923445701599, 2.6297638416290283, -0.23762765526771545, -1.2253801822662354, -0.40424445271492004, -0.02154519222676754, 0.9008948802947998, 0.9159940481185913, -0.7349756956100464, -0.8297791481018066, -0.21886250376701355, -0.25814884901046753, 0.6681212186813354, 0.05786646902561188, 0.6159993410110474, -0.7964404821395874, 0.8199850916862488, -0.3245046138763428, -0.9445955157279968, -1.9501609802246094, -2.895263433456421, -0.5398381352424622, -0.3965272903442383, -1.6360639333724976, -1.9447120428085327, -0.36738428473472595, 0.6118091940879822, 0.19062650203704834, 2.0047478675842285, -0.43785762786865234, -1.7406421899795532, 0.538936197757721, -0.14846794307231903, 1.5749108791351318, -0.8808534145355225, 1.3619471788406372, -0.8278377056121826, -1.2051900625228882, -0.6168670654296875, -1.7284255027770996, -1.0300081968307495, 0.28483468294143677, -1.757996916770935, -0.6577789187431335, -0.29920482635498047, 0.6305592060089111, 1.144449234008789, 0.17744538187980652, 0.5562980771064758, 0.9354936480522156, 0.8785611391067505, -0.10730026662349701, 1.9299907684326172, 1.0134861469268799, 0.5678386092185974, 0.012979025952517986, -1.5775936841964722, 0.0356585793197155, -0.9259161353111267, 0.9289919137954712, 0.28423553705215454, 0.1848224401473999, -0.7898216843605042, -0.4235180914402008, -1.2881503105163574, -0.7301387190818787, 0.2970917522907257, 0.8495899438858032, -0.5864861607551575, -0.8259982466697693, 1.145329236984253, -1.9043083190917969, -0.2887876033782959, 0.455424964427948, 0.7462840676307678, 0.46460315585136414, -0.9796565771102905, -0.9529938101768494, 0.25970447063446045, -0.7160426378250122, 1.1458837985992432, 1.0537950992584229, -0.9392752647399902, 1.1262152194976807, -0.616151750087738, 0.6606099009513855, 1.7268471717834473, -0.28790006041526794, -1.0076006650924683, -0.4122888147830963, 1.051368236541748, 0.20174413919448853, -2.5675110816955566, -0.4417691230773926, 0.15324419736862183, 1.499774694442749, 1.775849461555481, 1.2691822052001953, -0.3019898533821106, -0.8200145363807678, 1.0948666334152222, -0.14749178290367126, -1.4842333793640137, -1.1059318780899048, 1.5301949977874756, -1.1198011636734009, -1.6462138891220093, -1.9737364053726196, -1.4022648334503174, -0.4298378527164459, 1.0236833095550537, 1.3535614013671875, -0.7117483615875244, -1.1920320987701416, -0.687292754650116, 0.28548330068588257, 0.1351311206817627, 0.9049732685089111, -2.017061471939087, 2.018040895462036, -2.282590627670288, -0.408812552690506, -1.0735775232315063, 0.8066277503967285, 1.4877370595932007, -0.05463465675711632, 0.27823618054389954, 1.5529102087020874, 1.736811637878418, -1.3333450555801392, -0.44910168647766113, -0.023348985239863396, -0.6466782093048096, -2.6458709239959717, 0.10769779980182648, 0.09165707230567932, -0.4344644546508789, -0.041253793984651566, -0.8418391346931458, -1.6691166162490845, 1.1918271780014038, 1.0731401443481445, 2.884631872177124, 0.01808822900056839, -0.00015621018246747553, 0.26914462447166443, 0.975104808807373, -0.5858946442604065, -0.14099663496017456, 0.16775226593017578, 1.0768150091171265, -1.0798190832138062, -0.014544883742928505, -1.6455893516540527, 2.260863780975342, 1.2614243030548096, -0.614548921585083, 0.31900858879089355, 0.2313554286956787, -1.3015786409378052, 1.4724149703979492, 0.5477567911148071, -0.036055419594049454, 0.19059410691261292, -1.731704592704773, 0.13600072264671326, 1.8337385654449463, 1.4603376388549805, 2.5786287784576416, -0.7414077520370483, 1.4722613096237183, -0.0532732717692852, 0.5175474882125854, 0.8043010830879211, -0.4411078989505768, 0.05610746890306473, -1.2795578241348267, -0.8814493417739868, -0.41398561000823975, 0.9874811768531799, -0.6580967307090759, 0.45072993636131287, 0.7737000584602356, -0.797280490398407, 0.36721935868263245, -0.4266416132450104, -0.7556228041648865, -0.9666985273361206, -1.0557652711868286, -0.08353358507156372, 1.7234890460968018, -0.5066438913345337, -1.9465192556381226, -0.9320257902145386, 0.6610256433486938, -1.7496752738952637, 1.6782317161560059, -0.8539604544639587, 1.3994266986846924, 0.2618565857410431, -1.7265405654907227, -0.6558769941329956, 0.6340780258178711, 1.3476145267486572, 1.752793312072754, 0.18517877161502838, -1.0347909927368164, -1.892870545387268, 0.4603668451309204, 0.5405884981155396, 0.19218307733535767, -0.872097373008728, -0.2120080590248108, 2.010612726211548, -0.42413902282714844, -1.0623432397842407, -1.1574524641036987, -0.7847273945808411, 1.7627439498901367, 0.18572326004505157, -0.9710010886192322, -0.15757150948047638, -1.4154078960418701, -0.9389414191246033, 0.8586532473564148, -0.9536478519439697, 1.4026331901550293, -0.801718533039093, -0.22788755595684052, 0.8638091683387756, -0.14773938059806824, 1.3940098285675049, -1.521504521369934, 0.9079343676567078, -0.5736343264579773, 0.040092457085847855, 0.9808877110481262, -1.023071050643921, 0.9823772311210632, -1.4210011959075928, 1.0840511322021484, 0.5765699148178101, 0.05162734165787697, -0.23036813735961914, 0.44424912333488464, -1.2768160104751587, -1.4111088514328003, -0.40309855341911316, -0.16664613783359528, 0.4229128956794739, -1.3721333742141724, -0.04431099444627762, 0.7740611433982849, -0.053824882954359055, 0.5431603193283081, 0.3523256182670593, 1.5468273162841797, 0.8732019662857056, -0.3203423023223877, -0.9601350426673889, 2.357797622680664, -1.2296950817108154, 0.16502217948436737, 0.2509765923023224, 1.9139312505722046, 0.8976900577545166, -0.863908052444458, -0.6548640131950378, 0.15273533761501312, 0.913069486618042, 1.1760611534118652, -1.5295088291168213, -0.3901248574256897, -0.722743034362793, -2.0653278827667236, 0.4868890047073364, -1.1638554334640503, 0.059346266090869904, -0.20238618552684784, 0.3858749568462372, -0.23874981701374054, -0.9418673515319824, -0.5103363990783691, -0.22561690211296082, -1.7408713102340698, -1.496909260749817, 0.4453689455986023, 0.6625651121139526, -0.4433211088180542, 0.44764384627342224, 0.4197987914085388, 1.061331868171692, 0.16334712505340576, 0.446850448846817, 1.266608476638794, -0.6259452700614929, -0.258556991815567, -1.4347718954086304, -1.3279855251312256, -1.2335529327392578, -1.3729219436645508, -0.08100682497024536, -0.3216901421546936, 1.9436538219451904, 1.2808692455291748, -0.11205561459064484, 1.0009570121765137, -0.21549224853515625, -0.04315852001309395, 1.2050033807754517, 1.0537605285644531, -0.9417451620101929, -0.2791809141635895, -0.9524105787277222, -0.8650596141815186, 0.29248708486557007, 1.209139108657837, -0.5326158404350281, 0.11807545274496078, -1.3260328769683838, -0.34483739733695984, 0.4229625463485718, -1.2176827192306519, 0.6003639101982117, -0.5505265593528748, -1.3170374631881714, -1.0288807153701782, -0.10862318426370621, -0.47064757347106934, -1.3058183193206787, 0.6148233413696289, 0.342324435710907, -0.2907104194164276, -0.11936210840940475, 0.8187763690948486, -0.5408757925033569, 0.00835030060261488, 0.26851433515548706, -1.4107890129089355, -0.4524790942668915, 1.5564494132995605, 1.4695909023284912, 0.5362366437911987, -0.5271303653717041, -0.263479620218277, 0.016773030161857605, 0.31537503004074097, 0.17650234699249268, -0.5869859457015991, 1.067976474761963, -0.8607656359672546, 0.7631015777587891, 1.4559338092803955, 0.4287804067134857, -0.28625938296318054, -2.6311051845550537, 1.1936184167861938, 1.4053318500518799, -0.056618768721818924, 0.33227023482322693, 0.5392254590988159, 0.3061976134777069, -0.7595224380493164, -0.3928467631340027, -1.270014762878418, 0.9001995921134949, 0.6659714579582214, 1.98585045337677, -1.182450532913208, -0.19620610773563385, -0.11143781244754791, -0.3337741196155548, 0.13646583259105682, -0.11269709467887878, -0.6348387598991394, -0.21566730737686157, -0.5881375670433044, 1.2117100954055786, -0.49229657649993896, -1.3204714059829712, -0.2260860949754715, -1.197218656539917, 0.07512439787387848, 0.12357188761234283, -0.7278842926025391, 1.0523900985717773, -0.7627328038215637, 0.5464094281196594, 0.49573925137519836, -0.9952153563499451, 0.19806016981601715, 0.5484898090362549, -0.288287490606308, 0.5082020163536072, 0.36439135670661926, -0.7580965161323547, 0.9314829707145691, -0.8097376823425293, -0.17494820058345795, 0.2446538805961609, -0.9209794402122498, -0.3994826376438141, 1.0761418342590332, -0.7174903750419617, 1.4020599126815796, -1.466352939605713, 0.010600385256111622, 1.4958144426345825, -1.1131680011749268, 0.7792298197746277, -0.29731810092926025, -0.3659686744213104, -0.4261714816093445, -1.8619122505187988, -0.09007216989994049, -1.6904629468917847, -1.2163262367248535, 0.08176849782466888, -2.140324592590332, 0.7139079570770264, -0.8798079490661621, 0.3279481530189514, 0.10442864894866943, 1.2034924030303955, -0.08480630069971085, -0.27476558089256287, 0.845099687576294, -1.1445338726043701, -0.4713309109210968, -1.6362913846969604, -1.4373526573181152, -1.1232725381851196, 2.2524099349975586, 2.4082489013671875, 0.1320929378271103, -0.3303682804107666, -0.6551176309585571, 1.8003506660461426, 0.8380928039550781, -1.2607603073120117, 0.15172454714775085, 2.142789363861084, -0.5768862962722778, -0.22168096899986267, -0.27530795335769653, 1.6438074111938477, -0.18021364510059357, -0.6019604206085205, 0.8629665970802307, -0.8141658306121826, 1.6626849174499512, 0.3920193910598755, 0.4021841585636139, 1.1462870836257935, 1.168611764907837, 0.5211985111236572, -0.18360960483551025, 1.1529978513717651, -1.6535656452178955, -0.07286595553159714, -0.6018948554992676, -3.1850826740264893, -0.6950860023498535, 0.3605163097381592, 0.6465437412261963, 0.6029210090637207, -1.8374927043914795, -0.6065263152122498, 0.4224570095539093, -0.09379478543996811, 1.9483762979507446, -1.1695209741592407, -0.25443825125694275, 1.040144920349121, 0.405944287776947, 0.40857386589050293, -1.4146863222122192, -1.2453469038009644, -0.6436789631843567, -0.3571300506591797, -0.3116327226161957, -0.04493436962366104, 0.726892352104187, 0.37684422731399536, 0.4646076261997223, -0.5617304444313049, -0.573874294757843, -0.3196546137332916, 0.10583218932151794, 0.06260602921247482, 1.303484559059143, -0.7509085536003113, -0.4367200434207916, 0.20892246067523956, -0.19696733355522156, -0.25846704840660095, 0.8166622519493103, 0.728489339351654, 0.6935450434684753, -0.007912646979093552, 0.48240184783935547, 0.6600342392921448, 0.3006336987018585, -0.9930435419082642, -1.017773985862732, -0.008633793331682682, -1.0099542140960693, -0.8188064694404602, 0.43640458583831787, 0.6591635346412659, -1.6171598434448242, 0.13217675685882568, -0.035406142473220825, 0.25676429271698, -0.45860958099365234, -0.39318400621414185, 0.5316866636276245, -1.047277808189392, 0.5257636308670044, 0.4961681365966797, -0.39551419019699097, -0.615029513835907, 0.038718659430742264, 0.645556628704071, 0.3238248825073242, 1.1675302982330322, 1.5667304992675781, 1.5668644905090332, -0.966923177242279, 0.31754443049430847, -1.0868337154388428, 0.27661633491516113, -0.580205500125885, 1.0159653425216675, 0.6279154419898987, 1.283399224281311, 2.8670599460601807, -2.8346750736236572, -0.12539105117321014, -0.253776490688324, 0.3483615219593048, 0.11433950811624527, 0.2705041766166687, 0.8714864253997803, -0.230600506067276, -0.1721847802400589, -0.683174729347229, 0.06825036555528641, 0.5454875230789185, -0.7256373763084412, -0.05606833100318909, -0.2901877462863922, -1.0747114419937134, 0.5802562832832336, 0.7854791283607483, -0.02701530046761036, 1.241782784461975, 0.2694283425807953, -0.7085946798324585, -2.631322145462036, -0.4152401089668274, -0.44639503955841064, 0.34006771445274353], [0.2747352421283722, -1.5715162754058838, 0.39663010835647583, -1.066361427307129, -2.8237221240997314, 0.9209019541740417, 0.1327550709247589, -0.40475085377693176, -0.3800571858882904, 1.0418877601623535, -1.3605760335922241, -1.027356743812561, -1.1096727848052979, 1.1022136211395264, 0.5081177353858948, -0.11489449441432953, 0.7061768174171448, -0.6569194793701172, -1.0423524379730225, 0.9599167108535767, -1.072861671447754, -0.8256291151046753, -1.56143057346344, 1.1486451625823975, -0.5887597799301147, 0.7597559690475464, 0.33708420395851135, -0.6303848624229431, 0.9393934011459351, -0.06450194120407104, 0.724471926689148, 0.4152720272541046, -0.30458810925483704, -0.6735740900039673, -1.116087555885315, 0.7620155811309814, -0.04319137707352638, -0.9804452061653137, 1.2467771768569946, -0.16092677414417267, -0.5149791836738586, 0.38398829102516174, 1.5483970642089844, -0.7364452481269836, -1.8531826734542847, 0.5751078724861145, 0.019955476745963097, -0.3801290988922119, 0.5607262849807739, -1.5232127904891968, -0.30101388692855835, 0.22821247577667236, -0.8121591806411743, 0.8979605436325073, -0.0027319358196109533, -1.3947466611862183, -0.39520934224128723, -0.4326056241989136, 0.007151959929615259, 0.4038235545158386, 0.38131090998649597, 1.9164105653762817, 2.0418660640716553, 1.0462286472320557, -0.3980411887168884, -1.320983648300171, 1.403955101966858, 0.05671094357967377, 0.6523199081420898, -0.43254324793815613, 1.103031873703003, -0.5704027414321899, -0.34546783566474915, -0.2589680254459381, -0.2021854817867279, 0.25235509872436523, 0.19566094875335693, -0.39799046516418457, -0.26634615659713745, -0.4144361913204193, 0.8477921485900879, 1.0543828010559082, -0.0009899898432195187, -0.28664812445640564, -1.1506948471069336, -1.7135777473449707, 0.19022251665592194, -0.17822612822055817, -1.1200000047683716, -0.5354771614074707, -0.0035796973388642073, 0.13033874332904816, 1.776381254196167, -0.021896036341786385, -1.1090670824050903, -1.2661736011505127, 0.27640411257743835, -0.2969399094581604, -2.1907455921173096, 1.2212927341461182, 0.21036195755004883, 0.8236870765686035, -0.7108004689216614, -0.025189513340592384, 1.7727519273757935, 1.1724658012390137, 1.4003496170043945, 1.5639631748199463, 0.9819134473800659, 0.051106978207826614, -0.766471266746521, 0.26446112990379333, 4.528099060058594, -0.5473648309707642, -0.06171491742134094, 0.24994495511054993, 0.9005752205848694, -0.6621305346488953, -0.04632720723748207, 2.378188133239746, 1.3570541143417358, 0.2627449631690979, -1.2042906284332275, 0.42776286602020264, 1.4653527736663818, -0.07582750171422958, -0.2316378951072693, 0.32459092140197754, 1.543845534324646, 0.7959599494934082, -1.1510162353515625, -0.40200650691986084, -0.6927193403244019, -1.2378662824630737, -0.7157668471336365, -0.581606924533844, 0.254505455493927, -0.5901006460189819, -0.5468810200691223, -0.02249983511865139, 0.7334126830101013, -0.39492714405059814, -1.3086826801300049, -0.7732113003730774, 0.3780534267425537, 0.9857895970344543, -0.8240052461624146, 1.2957459688186646, 1.7460805177688599, 0.9840703010559082, -0.1761072874069214, -0.9585814476013184, -0.5982276201248169, -0.6098280549049377, -1.0165977478027344, -0.4497460722923279, -0.8595421314239502, -0.07068663835525513, 0.5270912647247314, -1.6199394464492798, -0.596868097782135, -0.6890312433242798, 0.16227658092975616, 1.0465278625488281, 0.5762686133384705, 0.31326207518577576, 0.5726742744445801, -0.9676131010055542, 0.2345767468214035, 0.20687884092330933, -1.0116175413131714, 0.2280246913433075, 0.49844977259635925, 1.9910709857940674, -0.2894696295261383, 0.26758506894111633, -0.23370207846164703, -0.2631383240222931, -1.0487736463546753, -0.4914109706878662, -1.4870761632919312, -0.8239482641220093, 1.6464641094207764, 0.5925680994987488, -0.36356088519096375, 0.8135951161384583, -1.0852253437042236, -0.02828109636902809, 2.066141366958618, -1.2257006168365479, 0.43271902203559875, -0.3017316162586212, 0.14545831084251404, -0.14128045737743378, -1.5727953910827637, 1.9212511777877808, -0.5274868011474609, -0.23876842856407166, 0.5538238286972046, 0.644307017326355, -1.68035888671875, -0.795757532119751, -0.1702987253665924, -2.42583966255188, 0.11270911991596222, -0.04508213698863983, 0.5975645780563354, 0.2077649086713791, 0.17120292782783508, 0.27548912167549133, -1.4312478303909302, -1.3671998977661133, -1.2518949508666992, -2.3404572010040283, 0.7697360515594482, -2.268510103225708, 0.19192321598529816, -1.6960968971252441, 0.7666382789611816, 0.0496959462761879, 0.02926958166062832, 1.2790285348892212, 0.2374524176120758, -1.320900559425354, 1.3521322011947632, -0.3056032955646515, 1.609988808631897, -0.7447879314422607, -0.25868573784828186, 0.6760613918304443, 0.8054176568984985, 1.2031936645507812, 0.8589191436767578, 1.5883809328079224, 1.55233633518219, 0.9857364296913147, 0.5131049752235413, 0.5103688836097717, 0.7834703922271729, -1.6245561838150024, -0.688304603099823, -1.347267746925354, -0.39729881286621094, -0.22568486630916595, 1.0327671766281128, 0.10954615473747253, -0.5932667851448059, -0.7667669653892517, 0.5646756291389465, -0.031397540122270584, -0.8650144338607788, 0.6216845512390137, -0.3678467571735382, 1.1159766912460327, -0.1379466950893402, -0.010189197957515717, 0.44085752964019775, 1.5182859897613525, -1.6728456020355225, -1.5846326351165771, -0.4072133004665375, 1.2781420946121216, 0.013818939216434956, 0.17249426245689392, -0.10687470436096191, -1.397630214691162, 0.7108827233314514, -0.19342520833015442, 0.5130883455276489, -0.7150583863258362, -0.7587121725082397, 1.180458426475525, -0.44047024846076965, 0.6454337239265442, 0.9139595031738281, -1.4365310668945312, 1.1323449611663818, 0.08160267025232315, -2.3133351802825928, -0.3252706825733185, -0.7016269564628601, 1.7096823453903198, 1.1657869815826416, 1.9240986108779907, 1.2308735847473145, 1.130040168762207, -0.7534153461456299, -0.8588849306106567, -2.6376194953918457, 0.14204610884189606, -0.06236693635582924, -0.29289761185646057, -0.7266058325767517, 1.5947542190551758, 0.5286194682121277, -0.578622579574585, 0.5938574075698853, 0.20746169984340668, -0.4606676697731018, 0.45059284567832947, 0.28030094504356384, -0.8169374465942383, -0.13353581726551056, -0.04518749937415123, 1.6148849725723267, -0.10625304281711578, 1.0483051538467407, 0.4438502788543701, 1.5384266376495361, 0.09901917725801468, 2.8182144165039062, -1.3372184038162231, -0.23375080525875092, 0.4733431339263916, 0.7891275882720947, -0.5157313942909241, 0.24417509138584137, 0.6511634588241577, 1.5706489086151123, 0.8293654918670654, 1.441846489906311, 0.4678781032562256, -0.9702507853507996, -0.46921300888061523, 0.6181798577308655, 0.001064202981069684, -0.005273886024951935, 0.24928295612335205, 0.16694758832454681, 0.15995587408542633, 0.25664207339286804, 0.5045489072799683, -1.6649513244628906, 0.3379359841346741, -0.6776567697525024, -1.2506482601165771, -0.015952954068779945, 0.9629379510879517, -2.316134452819824, -0.21030667424201965, 0.13430264592170715, -2.2390902042388916, -0.8470133543014526, -0.39377760887145996, -0.5199034214019775, 0.875892698764801, -0.07336746901273727, 0.4961390197277069, 0.575797975063324, 0.44057875871658325, 1.2908276319503784, 1.9311217069625854, 0.8626510500907898, 0.17046146094799042, -0.45738255977630615, -0.9842987060546875, -0.6740337014198303, -0.34648749232292175, 1.1383068561553955, -0.3182818591594696, 0.12041434645652771, 1.176568627357483, 0.7174041867256165, 0.09099540114402771, 0.18193483352661133, 0.8152327537536621, -0.8307231664657593, 0.2477046102285385, -0.7200594544410706, 0.031063780188560486, -0.5863063931465149, 1.2949328422546387, 0.36272719502449036, -0.3385457396507263, -1.0989502668380737, 0.07461177557706833, 0.9054735898971558, 0.06830354779958725, -2.0592141151428223, -0.35544338822364807, -0.3413890600204468, 0.2581612765789032, 0.1978921890258789, 0.5082757472991943, 0.26519525051116943, 0.8206202983856201, -1.2698469161987305, 0.01689152605831623, -0.7054096460342407, -0.057969823479652405, -0.29669177532196045, 0.47772231698036194, 0.641867458820343, -1.7030041217803955, 0.06932715326547623, -2.137014389038086, -0.917722761631012, 0.46996092796325684, 0.6133231520652771, 0.14839911460876465, -0.5928117036819458, -0.7043001651763916, 0.7719364166259766, -0.05457274615764618, -1.2621556520462036, -2.6704177856445312, -0.1883394420146942, 0.30255264043807983, -2.459839344024658, 1.5824111700057983, 0.6304231286048889, 1.0463393926620483, -0.6699048280715942, 0.1208510622382164, 0.6011061072349548, -1.0443376302719116, -0.33984947204589844, -1.6634916067123413, 0.3492553234100342, 0.42279529571533203, 2.2306737899780273, -0.5033565163612366, -0.7289785146713257, 0.5852304100990295, -1.5231233835220337, -0.9493889212608337, 0.47508320212364197, 0.4013362526893616, 1.8221406936645508, -1.0684322118759155, -1.0598770380020142, 0.2975975275039673, -1.181586742401123, -2.131183385848999, -0.6026066541671753, 0.8607448935508728, 0.9299806952476501, -0.7226256132125854, -0.9741036295890808, 0.1067657619714737, -1.8645374774932861, -0.9446126818656921, 1.1431149244308472, -0.2206648290157318, -1.747001051902771, -1.0233287811279297, -0.5070479512214661, -0.3817223310470581, -0.08060560375452042, -0.902556300163269, 0.14401020109653473, 0.4588204622268677, -1.1145586967468262, 0.1932554841041565, 1.4042904376983643, 0.28483474254608154, -0.6666746735572815, 0.256344199180603, -1.0694612264633179, -0.41615718603134155, 0.01076308824121952, -0.4219094514846802, -1.081505298614502, 1.1075447797775269, -1.5874502658843994, 1.4638559818267822, 1.1752634048461914, 0.17994001507759094, 0.21534018218517303, -0.13418473303318024, 0.8428825736045837, 0.44397449493408203, 0.4810653626918793, 0.26529422402381897, -1.5711283683776855, -0.4666357934474945, 0.4322364032268524, 0.33944761753082275, 1.0232292413711548, -0.223502516746521, -0.180852472782135, -1.317579984664917, -0.4328678548336029, 1.0945720672607422, -0.7846588492393494, -0.132017582654953, -2.191601037979126, 0.25805339217185974, 0.2400965690612793, -0.1861032247543335, -1.6458015441894531, -1.057261347770691, 0.805824875831604, -0.5786218047142029, -0.38615643978118896, 0.8670237064361572, -0.5172384977340698, 0.9025017619132996, 0.07657209038734436, -1.1158514022827148, 0.1262367069721222, -1.1927168369293213, -0.15967202186584473, -0.7704101204872131, 1.009202480316162, -0.13661538064479828, 0.44154515862464905, 0.23095828294754028, -1.1420791149139404, 0.5493685007095337, -2.312056303024292, -0.7030104398727417, 1.1318517923355103, 0.2872443199157715, -1.8689714670181274, 1.2503944635391235, -0.8083896040916443, -1.0701618194580078, 1.653591513633728, -0.29977676272392273, -0.4107392132282257, -1.1939032077789307, -0.543559730052948, 0.41950297355651855, 1.9177500009536743, -0.3169621229171753, -0.10783857852220535, -0.9298937320709229, -1.6713835000991821, -0.06455688178539276, 0.14402908086776733, -0.6204595565795898, 0.631580114364624, 0.31581535935401917, 0.132600799202919, -0.1888633370399475, 1.0991981029510498, 0.9422498941421509, 2.577348470687866, 1.932466745376587, -0.12694001197814941, -0.3124018609523773, -0.23010489344596863, -0.05776219442486763, -1.3830504417419434, -1.8291946649551392, -1.1685535907745361, 0.7142876982688904, -2.675248146057129, -0.6650707721710205, -1.1088554859161377, -0.8063421845436096, 0.5647501349449158, -0.8323883414268494, 0.16237857937812805, -0.4275156557559967, 2.4226670265197754, -0.08581352233886719, 0.5647186040878296, 0.018392767757177353, -0.3385154604911804, 0.5949903726577759, -1.0220005512237549, -0.09552726149559021, -0.06551650911569595, 0.5297324657440186, -0.9726383686065674, -0.9490318894386292, -0.47418496012687683, -1.4360028505325317, 0.051768958568573, 2.3080341815948486, -0.9437879920005798, 0.9449293613433838, 0.2947165071964264, 1.7026537656784058, -0.6035447120666504, -1.779922366142273, -0.6146323680877686, -1.6698201894760132, 0.911136269569397, -0.18406222760677338, -0.46383458375930786, 0.48738667368888855, -1.1114444732666016, 0.4752330482006073, -0.6573379635810852, 0.1256977766752243, -0.6734502911567688, 0.4054957628250122, 2.621453046798706, 1.2082555294036865, 0.8734456300735474, 1.0366623401641846, 1.35677969455719, 1.2062079906463623, 0.8746020793914795, -1.2704955339431763, 1.5686428546905518, 0.04612761363387108, -0.12808506190776825, -1.3921353816986084, 0.523612380027771, 0.22069592773914337, 0.1892150640487671, 0.3254474401473999, 0.550961434841156, -1.534791350364685, 1.8871583938598633, -0.7574371099472046, 0.08064468204975128, 0.5664638876914978, 0.610823392868042, -0.6060823202133179, -0.18116679787635803, 0.03496994078159332, -2.0373833179473877, 0.8053544163703918, 0.6291490793228149, 2.1288907527923584, 1.7862378358840942, 0.18569785356521606, 2.3733389377593994, 0.05941574275493622, -0.7756116390228271, 0.1571151316165924, 0.36425331234931946, 0.6486042141914368, -0.8676347732543945, -1.1362626552581787, 1.358392357826233, 0.035601671785116196, 1.2920317649841309, 0.033771585673093796, 1.3449469804763794, -0.20131030678749084, -0.0307808555662632, -0.1866472065448761, -1.4923100471496582, 0.7265682816505432, 0.1980307698249817, -0.1554373949766159, -0.24318447709083557, -1.7925734519958496, 0.8214755058288574, -0.2512892186641693, -2.61006760597229, -0.25885820388793945, -1.9063661098480225, 1.1486340761184692, 0.711338222026825, -2.069706678390503, -0.9075145721435547, 0.012475772760808468, -0.3459470570087433, -1.2003670930862427, -0.5934295654296875, 0.5843662619590759, -1.106019139289856, 0.9351005554199219, 0.717764675617218, 2.0202090740203857, 0.5746579170227051, 1.729636311531067, 1.1420954465866089, 0.8200594782829285, -0.3459573984146118, 0.013601982966065407, -0.5169364809989929, -1.0548053979873657, -0.7038385272026062, 0.18413609266281128, -0.8577828407287598, 0.8157011866569519, -0.5951441526412964, 0.03540642186999321, -0.028180068358778954, -0.06603702902793884, -0.8211561441421509, -1.0308607816696167, 0.9352630972862244, -0.013513528741896152, 2.364879846572876, -0.01618942618370056, -1.8201481103897095, -0.4495912790298462, -1.6597511768341064, 1.0011589527130127, -0.5890416502952576, 0.7811460494995117, 1.738860845565796, 0.2223903089761734, 0.2145162969827652, 0.2856127917766571, -0.8031604886054993, -0.36986473202705383, 0.80005943775177, -1.8296215534210205, -0.04846058040857315, 1.7682757377624512, 0.24398569762706757, 0.8978433609008789, -1.549232840538025, 0.22126919031143188, -0.8651995062828064, 2.0896565914154053, -1.2357286214828491, 1.0180760622024536, -0.5893309116363525, -1.0195436477661133, -0.8293412327766418, -0.523572564125061, -0.6660016775131226, 0.5028786659240723, -0.3934696614742279, 0.320615291595459, -1.5459953546524048, -0.28175631165504456, -0.010955951176583767, -0.4522067904472351, 1.4877766370773315, 0.05181273818016052, 1.341505527496338, 2.012573480606079, 0.041926298290491104, -1.0329327583312988, -0.4175379276275635, 0.46691858768463135, -1.5255502462387085, 0.6791440844535828, -1.9195094108581543, 1.0452370643615723, 0.038619063794612885, -0.676798939704895, -0.29244503378868103, 0.112189881503582, 0.29229629039764404, 1.7239694595336914, -2.0120184421539307, 0.1495516300201416, -0.6580609679222107, 1.6216877698898315, -2.792405128479004, 0.11248350888490677, 0.3371530771255493, -0.024509208276867867, -0.06578691303730011, -0.3664630055427551, 1.2656115293502808, 1.3412508964538574, 1.288353681564331, 0.3393727242946625, -1.5057889223098755, 0.003693582257255912, 0.12478924542665482, -0.01622169278562069, 1.4401872158050537, 0.533873975276947, -0.9328035116195679, -0.6116886138916016, -1.453771948814392, 1.7370879650115967, 1.4959826469421387, 1.4371259212493896, -0.21007314324378967, 1.6136314868927002, -0.45301783084869385, 0.285170316696167, 0.01018517930060625, 0.18895994126796722, 0.7704911231994629, 0.770155668258667, -1.5271821022033691, -0.034512460231781006, -0.4650816023349762, 0.5367650389671326, 1.9927643537521362, 0.0761418417096138, -0.2408500760793686, 0.36749398708343506, -0.5773681402206421, -0.5989954471588135, -0.48365065455436707, 0.18906374275684357, -2.031277894973755, 0.3944627046585083, -0.11658800393342972, -0.31195545196533203, -0.14426790177822113, -0.3442799746990204, 0.4883994460105896, 0.8711374402046204, -1.8540488481521606, 0.2903780937194824, -0.5914223790168762, -0.5852323770523071, 1.8613085746765137, 0.8660931587219238, 0.8840749859809875, 0.40712180733680725, -1.0072407722473145, -1.9476579427719116, 0.051731813699007034, 2.0780999660491943, -1.1102588176727295, 0.29968374967575073, 0.07342828065156937, -0.11833076179027557, -1.6732776165008545, -0.02812301740050316, -0.1311914324760437, -0.4372314214706421, -0.35694220662117004, 0.06123284995555878, 1.054787278175354, -1.0773686170578003, -0.9509587287902832, -1.2195677757263184, 0.6969624161720276, -2.080958604812622, -0.6867640614509583, 1.3693801164627075, 1.0081007480621338, -0.34011805057525635, -0.6594576239585876, 0.4856531620025635, -0.8394131064414978, -0.057695962488651276, -1.251795768737793, -1.663352608680725, -0.2275361865758896, -1.187495231628418, 0.5569999814033508, 0.1843366175889969, -3.0921356678009033, -1.0134357213974, 1.8508083820343018, 0.7913557291030884, 2.8667619228363037, 0.08017779886722565, -1.2405081987380981, 0.7052721381187439, -0.4899485111236572, -1.3805657625198364, 1.0908246040344238, 0.38605406880378723, 0.2992815375328064, 1.80801260471344, -2.2023444175720215, 0.6539175510406494, 0.6959123015403748, -1.682610034942627, 1.2445719242095947, -1.2920573949813843, -0.17187392711639404, -0.4074011743068695, 1.2073017358779907, -0.7388037443161011, -0.8735748529434204, 0.6701216697692871, 0.3305201530456543, -0.3412814736366272, -0.7473475933074951, -0.14120352268218994, -0.5340683460235596, 0.627963125705719, 0.08823644369840622, 0.021656813099980354, 0.2016616314649582, -0.4915536940097809, -1.153472661972046, -0.7987843751907349, -0.29216083884239197, -1.7722028493881226, -0.04668111726641655, -0.506627082824707, 0.3574577569961548, -0.497010737657547, -0.9403583407402039, 1.0974853038787842, -0.42028263211250305, -1.375827670097351, -0.7709831595420837, 0.3287920355796814, -1.352785587310791, 0.9783014059066772, -1.500785231590271, -1.1990076303482056, 0.3689987063407898, -0.5717056393623352, 0.6098632216453552, 0.10312783718109131, -0.9542528986930847, -1.0936717987060547, 2.284083366394043, -0.828725278377533, -0.5583553910255432, -1.851714015007019, -0.573697566986084, 1.444739818572998, -0.4658609926700592, 0.044338520616292953, -0.8944250345230103, -0.8483310341835022, 0.9151071906089783, 0.9585349559783936, 0.6949580907821655, 0.596796989440918, 0.26206329464912415, 1.2043590545654297, -1.0487524271011353, 1.6481728553771973, -1.1704126596450806, -1.1543477773666382, 0.8897722363471985, 0.9445759057998657, 1.6488919258117676, 0.09211380034685135, 1.2336442470550537, -0.3974166512489319, -0.44879940152168274, -0.8926771283149719, -1.520471453666687, 1.6326982975006104, -0.07439852505922318, 1.8030482530593872, -0.4621196985244751, 0.18626050651073456, 0.7817714214324951, -1.1229842901229858, 0.3302643895149231, -0.5040808916091919, -0.5131139159202576, 0.21636591851711273, -0.6280289888381958, -1.4368804693222046, 0.3037209212779999, 1.4902020692825317, -1.2857387065887451, 0.23192989826202393, 1.230545997619629, 0.051775697618722916, -0.24838677048683167, -0.3890337646007538, 0.5579243898391724, -2.3219947814941406, -0.15936417877674103, 1.0843549966812134, -0.4027692973613739, -1.116687536239624, 1.6170848608016968, 1.2344564199447632, 0.8454408049583435, 0.3017062246799469, 1.4126099348068237, -0.8046709299087524, -0.7838191986083984, 0.7654950022697449, 1.0885331630706787, 0.2048632651567459, -0.9909752607345581, -0.1180635392665863, -2.060927152633667, 0.5229699015617371, 1.030038595199585, -0.501446008682251, 0.18701577186584473, 1.2256749868392944, 0.8744609355926514, -0.3922298550605774, 0.20336385071277618, 0.3786879777908325, 0.08958959579467773, -0.7634917497634888, 1.0525131225585938, 0.274582177400589, -0.11632551997900009, 0.9587435722351074, -1.6096173524856567, 0.0027499543502926826, -0.8768853545188904, 0.4596981704235077, -0.17403662204742432, -0.7192109227180481, -0.41370129585266113, -2.2362899780273438, -0.45870184898376465, -0.31318363547325134, -0.9580084085464478, -0.497598797082901, -2.5610694885253906, 0.38056328892707825, 0.2776150107383728, 2.2238643169403076, -0.9068543314933777, 1.4092719554901123, -0.6846181750297546, 1.5876384973526, -0.1894347071647644, -0.8789272904396057], [0.6653140187263489, -3.2759037017822266, -1.1787816286087036, -0.5875270366668701, -0.5605406761169434, 1.2558754682540894, 0.4647124707698822, 0.42709624767303467, -1.9702397584915161, 0.8380990028381348, 0.7724612951278687, -0.24736745655536652, -0.18968795239925385, -1.0222935676574707, 0.7556647062301636, 0.3012895882129669, -0.6170680522918701, 1.5616397857666016, -1.0441811084747314, 0.08137936890125275, -2.1785192489624023, 0.22714489698410034, -1.747177004814148, -0.24649517238140106, -1.6310417652130127, -1.3935585021972656, 0.35230231285095215, -0.09009313583374023, 0.7155565619468689, 0.14642731845378876, 1.0617965459823608, -1.1381555795669556, 0.6969247460365295, 0.2241160273551941, -1.0821539163589478, -1.6353687047958374, -1.3164033889770508, 0.3410271406173706, -0.8549069166183472, -0.774869441986084, 1.470336675643921, 0.8695546388626099, -1.3834418058395386, -1.5634130239486694, 0.6330335140228271, 1.8228832483291626, 1.133437156677246, -1.1078224182128906, -1.4934149980545044, 1.2237154245376587, 0.43979355692863464, -0.3632234036922455, -1.108177661895752, -0.9969732761383057, -0.46110498905181885, -0.7588750720024109, -0.670078694820404, -1.0017364025115967, -0.4094180166721344, 0.3356165289878845, -0.17076483368873596, 1.4438811540603638, -1.4379537105560303, -0.8698294758796692, -0.6779597401618958, 1.364478349685669, 1.592569351196289, -0.4812394976615906, 0.8861101865768433, 0.4477650821208954, 1.2428505420684814, 0.09486168622970581, 0.34237873554229736, 1.6716015338897705, 0.6712148785591125, 0.16893362998962402, -1.166555643081665, -0.5703045725822449, 0.9760453701019287, 1.0851472616195679, -0.08303854614496231, 1.2416085004806519, 0.004367851186543703, 0.4460260272026062, -0.5648086667060852, 2.0298707485198975, 0.40782734751701355, 0.1682930290699005, -0.07730137556791306, 0.20123343169689178, 0.7264956831932068, 0.19746403396129608, 1.4930484294891357, 1.921041488647461, 0.6625503301620483, -1.2688038349151611, 0.03652124106884003, 2.0576353073120117, 0.5501890182495117, -0.6545595526695251, 1.4213277101516724, 1.2351298332214355, -0.03851025551557541, 0.6644005179405212, 0.32683172821998596, -1.3655771017074585, 2.552260637283325, -0.4809618294239044, -0.30608081817626953, 0.8892715573310852, 2.3357791900634766, -1.9249118566513062, -0.00332133611664176, 0.4402633607387543, 1.9316577911376953, -0.3182193338871002, -0.8371680378913879, -0.452157199382782, 0.17271865904331207, 1.0412321090698242, 0.5126670002937317, -0.7325825095176697, 1.6188091039657593, -0.39683595299720764, -0.37960338592529297, -0.33556288480758667, 2.2701663970947266, 0.21889442205429077, 0.4249227046966553, -0.5481521487236023, 0.7986307144165039, -0.8616434931755066, 0.3002222776412964, 0.45309978723526, -0.7027101516723633, 0.32526424527168274, 0.1526394635438919, -0.19121481478214264, 0.923373281955719, 0.7629517912864685, 0.5506743788719177, 1.2962563037872314, 0.1742505431175232, 0.24917636811733246, -1.0390734672546387, -0.460113525390625, -0.16562215983867645, -0.47016727924346924, 1.489797592163086, 0.01852835714817047, -1.6981539726257324, -0.3199811279773712, 0.045768383890390396, -1.9119313955307007, 0.6232866644859314, 1.704653263092041, 1.5249370336532593, -1.1198458671569824, 1.9778846502304077, -0.6250505447387695, -1.5133519172668457, 0.5182169079780579, -1.6401371955871582, -1.5372430086135864, -1.1530251502990723, 0.862864077091217, 0.8059313297271729, -0.20151536166667938, -0.9079384803771973, 1.2392712831497192, 0.739727795124054, -0.6165040731430054, -1.524040699005127, 2.112051010131836, -0.5313743352890015, 0.9199309945106506, 2.77270770072937, -1.2161730527877808, 1.2101370096206665, -0.6100457906723022, 0.5987197160720825, -0.024649545550346375, -0.1020653024315834, -3.1881446838378906, -2.448406219482422, -0.6951296925544739, -0.1762157380580902, 0.6245748400688171, -0.46265316009521484, -0.40802785754203796, -0.19676275551319122, -0.7154894471168518, -1.087436556816101, -0.9596808552742004, -0.455511212348938, -0.5928913354873657, -0.7248556613922119, 0.031064096838235855, 0.7807395458221436, 1.7581331729888916, 0.7599861025810242, 0.4866653084754944, -1.7553269863128662, 0.6983761191368103, 0.9598606824874878, 1.4599127769470215, 0.17593912780284882, -0.9087165594100952, 1.960181474685669, -1.0376667976379395, 1.9519014358520508, -0.018273405730724335, -0.9659863114356995, -0.07067859172821045, -0.8730925917625427, 0.6378980875015259, 1.742813229560852, 0.7417003512382507, 0.6962320804595947, -2.4158308506011963, 1.7804211378097534, 0.2102494090795517, -0.1019933819770813, -1.1110721826553345, -0.6201949715614319, -1.0392552614212036, 1.509294033050537, -0.20597364008426666, 0.5813263058662415, -0.4769374430179596, -0.506543755531311, 0.6241638660430908, 0.30809396505355835, -0.07624661922454834, 0.38720840215682983, 0.7485634684562683, 1.2404961585998535, -0.26461121439933777, 0.9270989894866943, -0.4288703501224518, -0.4983886480331421, 2.1294119358062744, 0.36645835638046265, -1.2763389348983765, 0.8324966430664062, 0.2713034451007843, -0.992795467376709, -0.3866860568523407, 0.22523394227027893, -1.807186245918274, 0.2050931602716446, 0.002115225652232766, 0.9608064293861389, 1.0856571197509766, 0.399176687002182, 0.9835801720619202, 0.43948692083358765, 0.2873584032058716, 0.3094642460346222, -1.4858214855194092, 0.7891826033592224, -0.21934297680854797, 1.0960969924926758, 0.09995561093091965, -1.211320161819458, -0.3664090633392334, -0.26118186116218567, -0.7984284162521362, 2.894765853881836, 0.6133800745010376, 0.5452911257743835, 0.755517840385437, 1.1525814533233643, 0.5714293718338013, 2.8632709980010986, -0.30070286989212036, -0.515053391456604, -0.2426343709230423, 0.25223785638809204, 0.3118222951889038, 0.07829922437667847, -0.8250125050544739, -0.13436277210712433, 0.32210519909858704, 0.014099306426942348, 0.5704144835472107, -0.014237562194466591, 0.49203065037727356, 1.1120861768722534, 1.9352062940597534, -0.9687658548355103, 0.10756649821996689, -1.7126743793487549, -0.12840856611728668, -0.4526478350162506, 0.42119044065475464, -0.061588749289512634, 1.3362741470336914, 0.6736571192741394, 0.3811626434326172, 1.2221592664718628, -0.7599076628684998, 0.04093950241804123, 0.25967058539390564, 2.3243203163146973, -0.7025310397148132, 0.6549809575080872, -1.401168942451477, -0.256612628698349, -0.13210415840148926, -0.784892201423645, 1.5319602489471436, 2.1225249767303467, 1.1799182891845703, -1.8496705293655396, 2.472831964492798, -0.029309572651982307, 0.9577406048774719, 0.16645516455173492, 0.04565737023949623, -0.3477080762386322, 0.3337518572807312, -1.62258780002594, 0.1289805918931961, 0.3848545551300049, 0.5896903276443481, -0.7715246677398682, 0.2542855143547058, 0.42748698592185974, 0.9163482785224915, 0.8216834664344788, -0.045069269835948944, -0.02945464849472046, 1.3893074989318848, 1.812006950378418, 1.5142086744308472, -1.350914478302002, 0.25219789147377014, 0.5116147398948669, 0.4294796884059906, 0.8573004603385925, -0.0878540426492691, 1.065557599067688, 0.3850100636482239, -0.8003702759742737, 1.1681146621704102, 2.1522037982940674, 0.841168224811554, -0.40308496356010437, -1.8052668571472168, 1.4131314754486084, 0.7085573077201843, 0.49681541323661804, 0.04733016714453697, 0.5550697445869446, 0.9047254323959351, -0.8101711273193359, -0.6454972624778748, -0.22124382853507996, 0.4010276198387146, 0.7166630625724792, 0.6233394742012024, 0.6846868991851807, 0.04322420805692673, -0.6291787028312683, -0.1853201538324356, -1.3060396909713745, -0.8299652338027954, -2.06961727142334, 0.7702249884605408, 0.9661646485328674, 1.0215651988983154, -0.366111695766449, 1.1043024063110352, 0.6171583533287048, -0.35247257351875305, -0.17421624064445496, -0.3607092797756195, -0.03921618312597275, 0.229551762342453, 0.3770058751106262, 0.7106847167015076, -0.11701023578643799, 0.7573708891868591, 0.1840975433588028, -0.5313616991043091, 1.8014789819717407, -0.7273930907249451, -1.108605980873108, 1.517749309539795, -0.9581322073936462, 0.4648233950138092, -1.212065577507019, 1.5461509227752686, -0.333726167678833, 1.0142567157745361, 0.17957323789596558, -2.65181040763855, 0.8919550180435181, -0.9022703170776367, 0.6024171113967896, -0.45594316720962524, 0.16184158623218536, 0.49888455867767334, 2.8524839878082275, 0.07161015272140503, -0.6843708157539368, -0.8915217518806458, -1.8706547021865845, -0.501434862613678, -0.17760802805423737, -0.02143721655011177, 1.654242753982544, -0.7517386674880981, -0.9822805523872375, 0.30996018648147583, 0.43456006050109863, -2.0658071041107178, -0.08386315405368805, 0.14111219346523285, 0.39730390906333923, -1.0497087240219116, 0.15017005801200867, 0.43639740347862244, 0.7762191295623779, 0.002253386890515685, 0.12211831659078598, 0.8544104099273682, -0.5703039765357971, -1.213854193687439, -1.108174443244934, 2.023015022277832, -1.5523852109909058, -1.2298966646194458, -0.2440527081489563, 0.7579118013381958, 1.086207628250122, 0.5146442651748657, -0.6214272975921631, 0.8652706742286682, -0.7347277998924255, -0.21146278083324432, 0.13351206481456757, 1.2554199695587158, 0.8943196535110474, 1.0669015645980835, -2.4134066104888916, 1.3298207521438599, 0.07098232209682465, -0.3105771541595459, -0.6097124218940735, 0.19383279979228973, -3.0423786640167236, 1.5037579536437988, 1.6938538551330566, 0.8986001014709473, 0.6342250108718872, 1.2521131038665771, -0.9190546870231628, 1.1367467641830444, 1.1682629585266113, 1.0351430177688599, 0.8644383549690247, 0.9182700514793396, -1.2942713499069214, -0.26271557807922363, 1.266007423400879, 1.543901801109314, 1.7293308973312378, 0.4979183077812195, -1.0490574836730957, -0.8666189908981323, 2.7214126586914062, -1.762593388557434, -1.3984555006027222, 0.39151591062545776, 0.2815034091472626, 2.8613040447235107, 1.2054190635681152, -0.9065613150596619, -0.5301028490066528, -0.7490296363830566, -1.3837617635726929, -1.2020949125289917, -1.832297444343567, -0.37810018658638, 0.9720914363861084, -0.9176777601242065, 0.481893926858902, -0.5274640917778015, -0.3894028961658478, -1.0408098697662354, -0.12663137912750244, 1.285605549812317, 0.629474401473999, -0.06050698831677437, 1.268182396888733, -0.03854791447520256, 0.05957181379199028, -0.4423351287841797, -1.080341100692749, -0.7152531743049622, 0.5576445460319519, -1.6138668060302734, 0.06912606954574585, 0.6278581619262695, 1.1757005453109741, -0.1334577202796936, 1.1399483680725098, 1.0620193481445312, -0.4677916169166565, -0.11774156987667084, 1.3680529594421387, 2.268242120742798, -0.8735920190811157, -2.057271957397461, -1.5107362270355225, -0.9182109236717224, 0.468440443277359, -0.49752864241600037, -2.2983999252319336, 0.27737659215927124, 0.26365926861763, 0.07593555003404617, -0.268051415681839, 0.9760565757751465, -1.200302004814148, -0.37361952662467957, 1.738953709602356, -1.1523436307907104, -0.14882288873195648, -0.1761784553527832, 0.14792503416538239, 1.3029096126556396, 0.28824833035469055, -0.4126616418361664, 0.2253018021583557, -0.9480178356170654, -0.8388245105743408, -0.6018637418746948, -0.22032153606414795, -1.193625807762146, -0.40553614497184753, -0.9535717964172363, 0.6802579164505005, -0.9459497332572937, 0.167949378490448, -0.9384728670120239, -0.9032581448554993, 0.04970955103635788, 0.8543328642845154, -0.878869891166687, -0.8190522789955139, 1.806492805480957, -1.1792768239974976, 0.740482747554779, -0.09313556551933289, 0.11681561172008514, 0.5637350082397461, 1.7862178087234497, -0.2328934669494629, 1.2373416423797607, 1.166254997253418, -1.358709454536438, 1.4639314413070679, -1.1646977663040161, -0.18450644612312317, -0.4483366906642914, -1.061050534248352, -2.335982084274292, -1.3207048177719116, 0.9129051566123962, 0.209712415933609, -0.4438384175300598, -0.6790496706962585, -0.697559118270874, -0.6465696692466736, 0.4058360159397125, 0.5461564660072327, 0.2712286114692688, -1.6621955633163452, -0.34966644644737244, 0.3863847553730011, 1.4591461420059204, -0.39939403533935547, -0.27329906821250916, 0.7983536720275879, -1.0813769102096558, -0.8971377611160278, 1.9103325605392456, -0.22326843440532684, -0.6606063842773438, 0.607881486415863, -0.2762224078178406, -1.1538002490997314, -0.24258896708488464, 1.9041160345077515, -0.6708327531814575, 0.5533804297447205, 0.22553256154060364, 0.4956527352333069, -1.2219586372375488, 1.0922809839248657, 0.8360683917999268, 1.4047346115112305, 0.4467516541481018, -0.11318591237068176, -1.0517127513885498, -1.3382203578948975, -0.21864360570907593, 0.6051406264305115, -0.25532791018486023, 0.8657100200653076, -0.4443505108356476, -0.27113252878189087, 1.3595298528671265, -0.6944662928581238, -1.04314386844635, -0.2965269684791565, -0.5572881698608398, 0.44818437099456787, 1.2007662057876587, -0.26852136850357056, 0.6470427513122559, 0.8157436847686768, 0.7489946484565735, 0.09087208658456802, -0.714225709438324, -1.244458556175232, -1.626702070236206, -1.9816592931747437, 1.8697903156280518, 0.5433071255683899, -0.6811546087265015, 1.612043857574463, 0.04124664142727852, 0.6083692908287048, 1.0996406078338623, 0.4950721263885498, 1.257340669631958, -1.3236992359161377, 0.4417547583580017, 0.13885930180549622, 0.36462658643722534, 0.20020100474357605, -0.04028285667300224, 0.8802167177200317, 2.1705219745635986, -0.44009825587272644, 0.9582212567329407, -0.11537644267082214, 0.553862452507019, -0.2519795298576355, 0.2215922623872757, 0.17453987896442413, -1.1533124446868896, -0.057873740792274475, 1.461186170578003, -0.6247990727424622, 0.9015837907791138, -0.02692735567688942, 1.8529984951019287, -1.7635254859924316, -1.285510778427124, 0.9746770262718201, -1.4594969749450684, 1.6189535856246948, -0.8076702952384949, 0.4792686998844147, -1.5233699083328247, 0.43773210048675537, -0.7649684548377991, -0.3397396206855774, -1.2568315267562866, 1.548142433166504, 0.02270001731812954, -0.12647926807403564, 0.002415600698441267, -0.5405815839767456, 1.892061710357666, 0.6401572823524475, 0.15440210700035095, 0.44281432032585144, 0.6443873047828674, -0.6778934597969055, -0.8244379162788391, -0.008318272419273853, 1.112656593322754, -0.8185245990753174, -0.5130899548530579, -0.5316728353500366, -0.32891833782196045, 2.133863925933838, 0.3093043267726898, -0.2848067283630371, -0.07712723314762115, 1.6550878286361694, 0.9846739768981934, 0.6993865966796875, -0.321900874376297, -0.4692322015762329, -0.884645938873291, -1.7358074188232422, 0.789888858795166, -0.4124329388141632, 0.016346165910363197, -0.5898130536079407, -1.111649513244629, 0.4991186261177063, 1.2283239364624023, -0.04435718059539795, 0.9438774585723877, -0.7850056290626526, -0.2858290374279022, 2.06927227973938, 2.0041685104370117, -1.436381220817566, 1.4321513175964355, 0.23680183291435242, -0.3713141977787018, 0.7202736139297485, 0.3001329004764557, 0.3297620415687561, -0.7275322079658508, 0.30535608530044556, -1.3772677183151245, -0.29037296772003174, -0.28537875413894653, 0.8740251660346985, 0.2744891941547394, 0.22315441071987152, -1.4961521625518799, 0.44387155771255493, 0.49965226650238037, -0.07802771031856537, 0.2491654008626938, -1.356975793838501, 0.019771995022892952, -0.3609410226345062, 1.4225459098815918, 0.44047388434410095, 2.709690570831299, 0.6710537672042847, 0.2898348867893219, -0.42852094769477844, -1.7654651403427124, 0.5843261480331421, -0.8124788403511047, -2.541724443435669, 0.05877448245882988, -1.2272772789001465, -1.2641472816467285, 1.6172025203704834, -0.010187997482717037, 1.7664462327957153, 0.7976797819137573, -0.4133094847202301, 1.0143146514892578, 1.2947450876235962, 0.1138550192117691, 0.5737236142158508, -0.6697453260421753, -0.8723469376564026, 2.7714316844940186, 0.09837505221366882, 0.2318485826253891, -0.4870348870754242, 0.03429100662469864, -1.0899779796600342, 0.41861492395401, -0.5756540298461914, -1.051544427871704, -0.881551206111908, -0.9161311388015747, -1.0506058931350708, 0.25694945454597473, -0.22163762152194977, -1.8840677738189697, -1.778388261795044, 0.39584505558013916, -1.6581941843032837, -0.7210900783538818, 0.3932461142539978, -0.02564993128180504, 0.8693593740463257, -0.46629583835601807, 0.5670414566993713, 0.5938529968261719, -0.08371739089488983, -0.33951830863952637, 1.6613409519195557, -1.2741007804870605, -2.4271340370178223, -0.49547091126441956, -0.39803674817085266, -0.5461317896842957, -3.4024431705474854, 0.8974737524986267, -0.42420050501823425, -0.753812313079834, 0.25143182277679443, -0.47024285793304443, -0.09609375894069672, -2.1872549057006836, -0.3786037862300873, -1.3634876012802124, -0.8940707445144653, -0.6074232459068298, 0.5810543894767761, -0.9067612886428833, 0.43064963817596436, 0.7435541749000549, 1.196918249130249, 0.43956923484802246, 0.5225597023963928, -0.1729862093925476, -0.22916579246520996, 0.5564014315605164, -0.9585438966751099, 0.01885545253753662, 2.5709033012390137, -0.8518858551979065, 0.7083045244216919, -0.35817739367485046, 0.4381670653820038, -0.005486877169460058, -0.48341891169548035, 1.0271869897842407, -1.90349280834198, 0.07081438601016998, 0.4306257665157318, -0.2512041926383972, -0.23879049718379974, -0.16618898510932922, -1.4825795888900757, 0.5486708283424377, 1.7177342176437378, -0.5352605581283569, -0.5295563340187073, -0.2674768567085266, -0.6564047932624817, 0.911371648311615, 1.7142698764801025, 0.21922144293785095, 0.5946996212005615, -1.0072559118270874, -1.6350582838058472, -0.755022406578064, 1.2435877323150635, -0.7568370699882507, 0.7919870615005493, 0.7955837249755859, 1.2590142488479614, 1.3438748121261597, -2.811931848526001, 0.16718757152557373, 0.18164584040641785, 1.3918125629425049, -0.9644896984100342, -0.455477237701416, 2.2200584411621094, 0.35889437794685364, 2.290231704711914, -0.9570331573486328, -1.929283618927002, 1.0828431844711304, -0.682428240776062, -0.8323537111282349, -0.1092766523361206, 0.5705132484436035, 0.775504469871521, -0.09176847338676453, 0.31672000885009766, -0.7797441482543945, 0.2814173102378845, 0.4135304391384125, -0.49861183762550354, 0.389242023229599, 0.1951119601726532, -1.0907875299453735, 1.1746169328689575, -0.33957117795944214, -1.1598740816116333, -0.8570137023925781, 0.9044889211654663, 0.5763401985168457, 0.29043254256248474, -0.21704769134521484, 0.5525850057601929, -1.4828294515609741, 1.3499277830123901, 0.9994875192642212, 0.26487278938293457, -0.8734527230262756, -0.4182984530925751, 0.6628946661949158, 0.6910164952278137, 0.19482338428497314, 1.4402016401290894, 0.18290507793426514, 0.6276447176933289, -0.4706684350967407, 0.2929410934448242, 0.22823059558868408, 0.38445910811424255, 1.3893147706985474, -0.819977879524231, 1.4462792873382568, 0.4973459243774414, -0.7145399451255798, 1.200840711593628, 1.357319951057434, 0.7548007965087891, -0.34186190366744995, -0.030717000365257263, 0.06670314818620682, 1.2360061407089233, -0.263264536857605, 0.9871453642845154, -1.1918338537216187, -0.6937476396560669, 0.1219513788819313, 2.3793041706085205, -1.1169198751449585, -1.3432462215423584, 0.30887454748153687, -0.25682780146598816, 0.47351667284965515, 0.8527318835258484, -0.2413366138935089, -0.8794381022453308, -1.0425677299499512, -0.607743501663208, 0.49076828360557556, -0.6404445171356201, -0.8572055697441101, -0.8262742757797241, 1.2639039754867554, -0.36036035418510437, -2.057544231414795, -0.5016615986824036, 0.9361851811408997, 0.22663068771362305, -0.6206637024879456, 1.114399790763855, 1.5032368898391724, 2.282186985015869, 0.5400609970092773, 0.23598720133304596, -0.2451082468032837, 0.44310393929481506, 1.844663381576538, -1.5507338047027588, -1.4336916208267212, 1.4161396026611328, 1.6291170120239258, 0.2626504898071289, -0.04565435275435448, 1.2847421169281006, -0.39728641510009766, 0.5005553960800171, 1.5423779487609863, 0.27825963497161865, 0.2533591091632843, 0.10734337568283081, 1.2078114748001099, -0.023566197603940964, -1.5027410984039307, -1.8209161758422852, 0.8269162178039551, 0.38765302300453186, 0.8334869146347046, -2.8593106269836426, 0.3428937792778015, 1.6782296895980835, -2.222038507461548, 0.4306189715862274, -1.1514109373092651, -0.5401790142059326, -1.3408045768737793, 1.1176735162734985, -0.7326687574386597, -0.4931548833847046, 2.5839791297912598, 0.5543655157089233, 0.1624433398246765, -1.2155545949935913, 0.677493155002594, 2.768998384475708, -0.6899412274360657, 1.2328592538833618, 0.0403057262301445, 1.0263577699661255, 1.548303246498108, 0.4518735706806183, 0.6742918491363525, 0.7974798679351807, -0.26213663816452026, 1.9060696363449097, 0.5543789267539978], [-1.3389856815338135, 0.07141824811697006, 0.37122607231140137, -0.5620158910751343, 0.4874689280986786, 0.8487169742584229, 0.29174375534057617, 1.7340195178985596, 0.3311026692390442, -0.18750470876693726, 0.14445355534553528, 0.574809193611145, 0.24407045543193817, 1.5748571157455444, -0.5476647019386292, 0.6744484305381775, 1.0480997562408447, -0.831892192363739, -0.2800067663192749, 0.14345212280750275, -0.5390263199806213, -0.651154637336731, 0.25528445839881897, -0.529449462890625, 0.6242361068725586, -0.7437646985054016, -0.7655271887779236, 0.1618347018957138, 0.5499083399772644, 0.29149723052978516, 0.6504536271095276, 0.016853883862495422, 1.0743181705474854, 0.15603910386562347, 1.2129749059677124, -0.5721049308776855, 0.015724821016192436, 0.39139267802238464, -0.14000774919986725, -1.4868589639663696, -0.5039132237434387, -0.18012148141860962, 0.879288375377655, 0.7233598828315735, 1.536181926727295, 0.5261237025260925, -0.3492657542228699, -1.3523318767547607, -0.3984143137931824, 0.7751617431640625, 0.3497232496738434, 0.3161110579967499, -1.018755316734314, 1.1686521768569946, 0.7634015083312988, 0.4073675870895386, 0.15758894383907318, -1.434715986251831, -1.4213075637817383, 0.7583489418029785, -0.2565523386001587, 0.4590304493904114, 0.5244531631469727, -1.0411443710327148, 1.6376854181289673, 1.0216621160507202, -0.12904009222984314, -0.8112831711769104, 0.01604948565363884, -0.43615663051605225, 0.10061818361282349, -0.22272132337093353, -0.35647091269493103, 0.3701465427875519, -0.7638804316520691, 1.1633168458938599, 1.1683592796325684, -1.1341520547866821, 0.6560880541801453, -0.4413277804851532, 0.8237267136573792, -1.0810056924819946, 0.7277446389198303, -0.24561618268489838, 0.3796994388103485, 0.9930242896080017, 0.09688006341457367, 0.9890801310539246, -0.10598664730787277, -0.15714381635189056, -1.0672388076782227, 0.4461987316608429, 0.8915403485298157, -0.5238789319992065, 0.8839012384414673, -0.3536188304424286, 1.6310042142868042, 0.5882189273834229, -0.8131844401359558, -0.44394320249557495, 0.9577182531356812, -0.7000404000282288, 0.5258699655532837, -0.4771181046962738, 0.4406589865684509, -1.38503897190094, -0.1814012974500656, -0.6760832667350769, -0.4790396988391876, -0.8888053297996521, 0.05895806476473808, -0.32705551385879517, -1.863373875617981, -1.2830023765563965, -0.5447397828102112, 2.637047052383423, 0.820248544216156, 0.4071250557899475, 0.23695604503154755, -0.7143684029579163, -0.5548110604286194, -1.3229749202728271, -0.1265760362148285, 0.6664904356002808, 0.30348658561706543, 1.5393704175949097, 2.2033824920654297, -1.4581794738769531, 0.16533513367176056, 0.7921120524406433, -0.4572386145591736, -0.24450623989105225, -0.4630470871925354, 0.19668258726596832, 0.09815730154514313, -0.6033609509468079, -0.1759224534034729, 0.14981992542743683, 0.33143171668052673, -0.40242817997932434, -1.5466364622116089, 1.4256267547607422, -0.3898407518863678, 0.030426036566495895, -0.06307920813560486, 0.022822486236691475, -1.4980831146240234, -0.48818013072013855, -1.4012656211853027, 1.4344909191131592, -1.6347362995147705, 0.032655034214258194, 1.9228825569152832, -0.7015183568000793, 2.5627007484436035, 0.47304096817970276, 0.1915811449289322, -0.6257995367050171, 1.835057258605957, -0.032303500920534134, -0.7324961423873901, 0.43250569701194763, -1.05222749710083, -0.6497929096221924, -1.0763782262802124, -1.8655129671096802, -0.888221025466919, 0.49307936429977417, -0.6993638873100281, 1.1597256660461426, -0.041837312281131744, 0.06396220624446869, 1.8447901010513306, -0.3621601462364197, -0.030345255509018898, -0.012834899127483368, -0.8167586326599121, 0.5045657157897949, 0.1559632569551468, 1.0940747261047363, -0.08194214105606079, -0.08568555861711502, 0.37976911664009094, 0.6460665464401245, -1.8049702644348145, 0.9850090146064758, -0.7673841714859009, -0.6746307015419006, 1.070778250694275, 0.9715349078178406, 0.34304291009902954, -1.007346272468567, 0.8938677310943604, 0.42909693717956543, 0.8410179018974304, 0.23938967287540436, 1.1478201150894165, -0.34701818227767944, 1.5555970668792725, -0.37607699632644653, -0.896994948387146, -0.5830869674682617, 0.26581069827079773, -0.5773043632507324, 2.5336737632751465, -0.44483402371406555, 1.145308017730713, -1.093220829963684, -1.9279143810272217, 0.27926069498062134, -1.2242077589035034, 0.5396543741226196, -1.1144148111343384, 0.4736331105232239, 0.11439167708158493, -1.4367742538452148, 0.7602621912956238, -0.117156021296978, 1.7587567567825317, 0.534080982208252, 0.627578616142273, -1.198391318321228, 0.6974188685417175, 0.10542916506528854, -0.4041925072669983, 1.0507875680923462, 0.2617175877094269, 0.8582590818405151, 0.5727453827857971, 0.6339438557624817, 0.9896792769432068, -0.0868181437253952, -0.5480989217758179, 0.7912032008171082, -0.40462929010391235, 0.47097694873809814, 1.0037034749984741, -1.1003620624542236, 0.9629514813423157, -1.2029134035110474, 0.08458102494478226, -0.20301786065101624, -1.2500258684158325, -0.8001523613929749, -1.5879004001617432, 0.7218127846717834, -1.100258231163025, -1.712011456489563, -0.700795590877533, 0.037012092769145966, -0.31843653321266174, -0.22583705186843872, 1.3277684450149536, -0.17281731963157654, -1.2224152088165283, -1.5678009986877441, 0.26813527941703796, 0.8753563165664673, -0.764665424823761, -0.15875349938869476, -0.9583292603492737, -1.0602538585662842, -1.5128473043441772, -0.5403291583061218, -0.5014292001724243, -0.10618210583925247, 0.8087253570556641, 0.5956453084945679, -0.6938188672065735, -0.6351234316825867, 3.6545045375823975, -1.4439287185668945, -0.25639796257019043, 0.8227496147155762, -1.0497466325759888, -0.9097151160240173, 0.35602372884750366, -0.06692267209291458, 0.5323976874351501, -0.5536643862724304, 0.8271318078041077, 0.9779579639434814, 0.8641343712806702, 2.0515334606170654, 0.32142677903175354, 0.9789279103279114, -1.5036234855651855, -1.7829679250717163, 0.49360963702201843, -0.10681542009115219, 1.007216215133667, -0.29867294430732727, 0.47827795147895813, 0.6993534564971924, -0.40062347054481506, 1.7299466133117676, 0.8632400631904602, -0.5188496112823486, 0.7220160365104675, -0.789903998374939, 1.3634556531906128, 0.34042009711265564, -0.37283921241760254, -0.8880863785743713, -0.6544814705848694, -0.3112649917602539, -0.28925666213035583, 0.9128548502922058, -1.2494014501571655, -0.5340626239776611, -1.6868520975112915, -0.9170207977294922, 0.243898406624794, 0.4457174837589264, -1.6249687671661377, -1.7534737586975098, 0.3258424401283264, -0.8476507663726807, 0.4409147799015045, 0.7490931153297424, -0.6334284543991089, -0.8312855958938599, -0.2596563994884491, -0.9770264625549316, -0.42573726177215576, -0.8772796392440796, 0.23961323499679565, 0.23142454028129578, 0.0265517458319664, 0.4031783640384674, -0.9292460083961487, 1.726276159286499, 1.5515620708465576, -0.17648805677890778, 0.5041124820709229, 1.0181649923324585, -0.3141331374645233, -0.2748218774795532, 0.6076909303665161, 1.0696830749511719, -0.955377995967865, -0.25174427032470703, -0.6433267593383789, 1.8951340913772583, -1.2624685764312744, 0.3738151490688324, 0.3517579138278961, 0.9563905000686646, 0.6163815259933472, -0.6350777745246887, 0.4692700505256653, -1.6114646196365356, 0.8873435258865356, 0.34222933650016785, -0.14005199074745178, 1.2984071969985962, -0.8153167366981506, -0.4378899335861206, 1.1100136041641235, -1.8749607801437378, -1.4560141563415527, 0.03757321089506149, -0.1739087849855423, -1.517307162284851, 1.0583921670913696, 0.9598608613014221, 0.49386143684387207, 0.5641010999679565, 2.295834541320801, -0.4223703145980835, 0.3953622281551361, -0.5419747233390808, -0.43613114953041077, -0.36800292134284973, 0.8131293058395386, -1.6036609411239624, 1.8258683681488037, -0.44004538655281067, 0.9258294105529785, 1.4665272235870361, 1.0781697034835815, 0.6800140738487244, -1.0152586698532104, -0.23296231031417847, 0.677893877029419, -1.1538249254226685, -0.08221051096916199, -0.4786030352115631, 1.606022834777832, 0.7001023888587952, -0.4037758409976959, -1.2690421342849731, 0.12217433750629425, -0.8843285441398621, -0.08750486373901367, -1.2604957818984985, -0.5640835762023926, -1.1461416482925415, 0.20967550575733185, 0.10264238715171814, -1.993618130683899, 1.7198492288589478, -1.1251497268676758, -0.6516932845115662, -1.4523742198944092, 1.3658764362335205, -1.11669921875, 0.7179707884788513, 0.7036399245262146, -0.46461230516433716, 0.8973227739334106, -2.158907413482666, -0.45613357424736023, -1.758130431175232, -0.8821253180503845, 1.70339035987854, 1.2190290689468384, 0.3320518434047699, 0.28025245666503906, 1.8850045204162598, 1.1909465789794922, -0.5526818633079529, 0.6203141808509827, 1.2568199634552002, -0.2438773661851883, 0.3734855055809021, 0.5388668775558472, 0.7946230173110962, -0.3419228792190552, -0.3844951093196869, -1.5261679887771606, 0.563836395740509, -1.7649481296539307, 0.06677424907684326, 0.44356876611709595, -0.16982541978359222, -0.9747982025146484, 0.6648755073547363, 1.1598871946334839, -0.22076527774333954, 0.8568893671035767, -0.9130309224128723, 0.26870909333229065, 0.5039705038070679, -0.2369457185268402, 0.48243874311447144, -1.5934456586837769, 1.5935306549072266, -0.44357046484947205, 0.14762501418590546, -2.313765525817871, -0.6849156618118286, -0.4626518785953522, -0.148110494017601, -0.7617818713188171, 0.17924846708774567, -0.4713347256183624, -0.6290545463562012, -0.3023841381072998, -0.13163046538829803, 1.070876121520996, -1.2244967222213745, 0.4217594265937805, 1.4363387823104858, 1.0948739051818848, -0.2822338044643402, 0.2503948509693146, 0.8423480987548828, -1.367169976234436, 0.8187119364738464, -0.7992361187934875, -0.6515666842460632, -0.16937290132045746, -0.048156045377254486, 1.2839688062667847, 0.22692330181598663, 0.057389672845602036, 1.2810351848602295, -1.1738489866256714, -0.3327978551387787, -0.753389835357666, -0.39403489232063293, 0.047173574566841125, -0.03201829642057419, 2.6602470874786377, -0.569698691368103, -0.27760276198387146, -0.7716040015220642, 0.8815982341766357, -0.7113020420074463, -0.7179627418518066, -0.41917452216148376, 0.18540945649147034, -0.5329447388648987, 0.22586674988269806, 0.3248191177845001, -0.9114817976951599, 0.5317498445510864, -1.3477418422698975, -0.4655071198940277, -0.03201751038432121, 2.792090892791748, -0.9185317754745483, 1.2246119976043701, -0.27998101711273193, -1.5706712007522583, 0.41138753294944763, 0.1413717269897461, 1.423938274383545, -1.6777775287628174, -0.6726048588752747, 1.4456590414047241, 0.0718759223818779, 1.091371774673462, -0.2844756543636322, 1.0884099006652832, -0.2154223769903183, 0.20120832324028015, -0.35309526324272156, -1.0920957326889038, -0.4475751221179962, -1.1296436786651611, -0.19734595715999603, -1.184019684791565, 0.6191976070404053, 0.8149452209472656, 0.35620903968811035, -0.4386627674102783, 1.723706603050232, 0.07874419540166855, -0.6523458361625671, -1.383902907371521, 0.06512793153524399, -0.7795573472976685, 0.018100298941135406, 2.1773149967193604, -1.7531418800354004, -0.34441065788269043, -0.42508596181869507, -0.23151268064975739, -0.26177600026130676, -1.2340322732925415, -0.04610295593738556, -0.8631006479263306, -0.1450241357088089, -0.7599934935569763, -1.057794213294983, -2.347471237182617, 1.2986600399017334, 0.8590212464332581, 1.2130706310272217, -1.2086920738220215, -1.0498210191726685, -1.376110553741455, 0.41412603855133057, -1.8889107704162598, 1.6665327548980713, -0.7722508311271667, 0.5016369223594666, -1.5204620361328125, -0.6762974262237549, 0.03982318565249443, 0.17288514971733093, -0.8694491982460022, -0.13832387328147888, -0.5383340120315552, 2.0260422229766846, 0.5089855194091797, 0.630169153213501, -0.654385507106781, -0.21257241070270538, 1.4672963619232178, 0.14192970097064972, 0.9363475441932678, 0.4843825697898865, 0.4830925166606903, 0.2502492070198059, -0.36086273193359375, 0.5221124887466431, -0.7339433431625366, 0.813656747341156, -0.25303563475608826, -0.13047738373279572, 1.657570242881775, -1.3792369365692139, -1.1428457498550415, -0.7759315371513367, -0.8280497789382935, -1.5602176189422607, -0.7846124172210693, -0.35349714756011963, 2.095612049102783, -0.6983758807182312, -1.7957472801208496, -0.9935098886489868, 2.7179415225982666, -1.9182313680648804, 0.2541595995426178, -0.1241014152765274, 0.3860056698322296, -1.0792148113250732, 1.4168843030929565, 0.10623510181903839, 0.7635831832885742, -1.103320837020874, 1.5660468339920044, -1.5139166116714478, 0.7193096876144409, 0.6477003693580627, -0.7675926089286804, -1.4402992725372314, 1.054105520248413, -0.12005815654993057, 2.51967191696167, -0.4658309519290924, 1.1934044361114502, 2.3971147537231445, 0.12797099351882935, 0.9360376596450806, 1.2465715408325195, -0.9733068346977234, -0.400945782661438, -0.7245347499847412, 0.6087667346000671, -1.0419574975967407, -0.9941193461418152, -0.1576734483242035, 0.0045476206578314304, 0.7285580039024353, -1.1512446403503418, -0.5803140997886658, -0.1518869251012802, -0.7681871056556702, -0.2352684736251831, -0.03625034913420677, -0.14473244547843933, -0.8736667037010193, 0.8979460000991821, -0.5118810534477234, 1.100062608718872, 1.1430548429489136, -1.3800827264785767, 1.802454948425293, 1.4254142045974731, -0.8275171518325806, -0.8461654186248779, 0.3741832375526428, -1.564599871635437, -0.7850694060325623, 0.23531371355056763, -0.008252006955444813, 0.904103696346283, 0.5717993974685669, -0.009411728009581566, -0.5312141180038452, 0.28549063205718994, -0.6189544796943665, -0.2843257486820221, 1.40513277053833, 0.8794937133789062, -0.9982858896255493, -0.13404236733913422, 1.8410214185714722, 1.320512056350708, 1.4419184923171997, -1.8085342645645142, -1.1436954736709595, -0.9861688017845154, 0.11179624497890472, -0.9216994047164917, -1.1894774436950684, 0.4345437288284302, -0.9202832579612732, 0.5181456804275513, 0.3335256278514862, 0.6587130427360535, -2.525219440460205, -0.8469070792198181, 0.1605534851551056, -1.4238814115524292, 0.515470027923584, -0.18364715576171875, 0.5927519202232361, -1.4228837490081787, -0.2875743508338928, -0.9649417996406555, 0.4815314710140228, 1.7731703519821167, 1.8659133911132812, 0.2629724442958832, 0.36997586488723755, 0.6034106612205505, -1.1754705905914307, -0.13566111028194427, -0.17191006243228912, 0.5795065760612488, -0.03791322559118271, 0.31553152203559875, -0.20559395849704742, -0.3466578722000122, 0.34660741686820984, -0.25982996821403503, 0.4030613899230957, -0.8505783677101135, 0.10952527076005936, -0.7412058115005493, -0.06291978806257248, 0.19990478456020355, -0.6985172033309937, -0.9486731886863708, -0.8133881092071533, 0.40180933475494385, 0.6072019934654236, 1.2638167142868042, -1.0433745384216309, 0.04144391790032387, 0.5002108216285706, 0.7062760591506958, 0.8458139896392822, -0.4340236485004425, 0.9538764953613281, 0.1652967631816864, 0.7938051819801331, -0.057936206459999084, 0.627017617225647, 0.15518610179424286, -1.542018175125122, -0.8458916544914246, -0.41311922669410706, -0.02465633861720562, 1.1499807834625244, -1.1129957437515259, -1.1081925630569458, 0.7778357863426208, -1.047042727470398, -0.9054626226425171, -1.4651004076004028, -0.46886971592903137, -1.8133310079574585, 0.9229480028152466, -0.9916163086891174, -0.1355687826871872, 1.6344102621078491, 0.724578320980072, -0.785995364189148, 0.9576352834701538, -0.1281728744506836, -0.13619087636470795, 0.1926700919866562, -0.5920110940933228, 0.878276526927948, -0.0718315839767456, -0.45237842202186584, -1.4719960689544678, -0.4251477122306824, 0.35420069098472595, 0.540000319480896, 1.2174793481826782, -1.0819385051727295, 1.306330919265747, 0.11111792176961899, 0.697217583656311, 0.6709890365600586, 0.7486141324043274, -0.04733799770474434, -0.11781343072652817, 0.20853495597839355, -1.5770989656448364, -0.17899401485919952, -0.36729681491851807, 0.24896878004074097, -1.3926581144332886, -0.12167003005743027, 0.5158858299255371, 1.6037282943725586, 0.6059061288833618, -0.179937481880188, 0.5927020311355591, -0.1525336652994156, 0.4609048664569855, 0.35014691948890686, 1.9252995252609253, 2.689263343811035, -0.5395178198814392, -0.3254295885562897, 0.838394284248352, 0.6594832539558411, 0.62326979637146, 1.8555870056152344, -0.5499058961868286, -0.4287521243095398, -0.8672704100608826, -0.42956802248954773, 0.4693804085254669, 0.5837408900260925, 2.121180772781372, -0.9798105359077454, 0.8854969143867493, -0.3764079511165619, 1.2514824867248535, -0.07131654769182205, -0.8471415042877197, 1.8389670848846436, 0.8202831149101257, 0.1066749095916748, 1.7926570177078247, 0.6394784450531006, 0.24378283321857452, 0.3306727409362793, -1.0808589458465576, -0.7925599217414856, -0.03670885041356087, 2.2036094665527344, 2.3547871112823486, 1.0686380863189697, -0.8621094822883606, -2.860426187515259, 0.40538451075553894, -1.9582873582839966, 0.06781048327684402, 0.10847540944814682, -0.15271329879760742, -0.39177510142326355, -0.9874116778373718, 0.8944041132926941, -0.007593767251819372, 1.1288241147994995, 0.023678360506892204, -0.9831333160400391, 0.715524435043335, 0.6647661328315735, 0.8603225946426392, -1.9161036014556885, 0.7380024790763855, 1.4524292945861816, 0.7807742357254028, -0.02529144659638405, 0.19635885953903198, -0.5196585059165955, 0.09693990647792816, -0.5178092122077942, 1.1221646070480347, -0.24265238642692566, -0.08258455246686935, -0.6735198497772217, -0.6143296360969543, 0.015742741525173187, 0.24506963789463043, 0.03589072450995445, -0.3031995892524719, 1.5386245250701904, -0.9088364839553833, 0.30680957436561584, 0.2823789417743683, -0.23387983441352844, -0.054692912846803665, 0.1885894536972046, 1.8350993394851685, 0.6334384083747864, -2.311121940612793, -0.6243067383766174, 0.503572404384613, 1.672895908355713, -1.3736292123794556, 1.2397381067276, -0.4716123044490814, -2.0391736030578613, 0.0856531411409378, 1.7054438591003418, 0.2651154100894928, 0.7175903916358948, -1.0847870111465454, 0.11235353350639343, -0.6535214781761169, 0.27277907729148865, 1.3176223039627075, -0.40230023860931396, -1.2651152610778809, -1.9598560333251953, 0.983733594417572, -1.8049484491348267, 0.32011404633522034, 0.5988112092018127, -0.13383930921554565, 1.7205449342727661, 0.3648245632648468, -0.6177648901939392, -0.2642115354537964, 0.2588241696357727, -1.3814373016357422, 0.8653422594070435, -0.3093428611755371, 1.6611638069152832, -0.8563734292984009, 1.859047532081604, 1.2636637687683105, 1.7973322868347168, -0.1734662801027298, 0.2565367519855499, -0.6955748796463013, -1.4778028726577759, 1.0691756010055542, -0.08868959546089172, 1.3585313558578491, -0.9723363518714905, 0.7569658160209656, -1.5147489309310913, 1.1016815900802612, -0.7069809436798096, 1.8942151069641113, 0.3279566764831543, 0.25654253363609314, -1.3739595413208008, 0.1091965064406395, 0.22020097076892853, -1.9555916786193848, -0.771622896194458, 0.5795787572860718, -0.02837945893406868, 0.32724860310554504, -1.9831011295318604, 0.14866295456886292, 0.12416800856590271, -0.7315869331359863, -0.9439712762832642, 0.32762885093688965, -0.494078665971756, 1.6690902709960938, -0.6577025651931763, -1.0377655029296875, 1.48403799533844, -0.15512172877788544, 0.02466898038983345, 0.26903036236763, 0.7373945116996765, 1.5292004346847534, -1.4777467250823975, -0.9624742269515991, -0.3321616053581238, 0.13341695070266724, -0.6044629812240601, 0.13686016201972961, 1.1657627820968628, 0.3410431742668152, -1.380246639251709, 0.5506439805030823, -1.7021218538284302, 0.1895066797733307, -1.1580171585083008, 0.617702066898346, 0.6478862762451172, 0.7805115580558777, -0.8103960752487183, 1.0946476459503174, 0.37511569261550903, -0.4352729320526123, -0.4119781255722046, 1.8740198612213135, 1.2930883169174194, -0.18383991718292236, 0.21968451142311096, -0.37854722142219543, -1.1470037698745728, -0.7088143825531006, 1.8040494918823242, 1.7330403327941895, 1.8211050033569336, 0.6667999029159546, 0.058732155710458755, 0.21499229967594147, 1.0227042436599731, 0.9898709058761597, 0.711127758026123, -0.5867255330085754, -1.4003117084503174, 0.5915313363075256, -0.05825451388955116, -0.9512874484062195, 0.37432777881622314, -0.04984413459897041, -0.3538450598716736, -0.7311168313026428, 0.3523755669593811, -0.376375287771225, -0.9514910578727722, -0.3891845941543579, -0.9398230314254761, 1.2991890907287598, -0.9040823578834534, -0.6083871722221375, 0.7568459510803223, 1.870312213897705, -1.01410710811615, -1.4311798810958862, -0.17613866925239563, -1.0567758083343506, 1.2603654861450195, 0.13802409172058105, 2.06850004196167, -0.8544338941574097], [-1.0813614130020142, -2.6560583114624023, 0.9147379994392395, -1.4190999269485474, 2.7937099933624268, 0.9662119746208191, -0.012346011586487293, -0.2748475670814514, -0.09565110504627228, 0.00976366177201271, -0.1914430409669876, 1.1169095039367676, -0.38120079040527344, 0.6911590099334717, -0.8566555380821228, 1.2211568355560303, -0.5815926194190979, 0.6729851365089417, -1.344648003578186, -0.1398286074399948, -1.6443185806274414, 0.48888278007507324, -1.2923798561096191, 0.8986998796463013, 1.4954782724380493, 0.5727301836013794, 2.476827383041382, -0.7983734607696533, 0.8032050132751465, -0.1980319321155548, 0.8694624304771423, -0.05818140506744385, -0.8080138564109802, -1.2001217603683472, -0.671501100063324, 1.1153967380523682, 0.14134986698627472, 0.32160744071006775, -0.695215106010437, 0.870932936668396, 0.5338497757911682, -0.6727134585380554, 1.718084454536438, 1.5075093507766724, -0.7914236187934875, -0.0912993997335434, -0.22064208984375, -0.35472118854522705, -0.09942441433668137, 2.8608596324920654, 1.1970384120941162, -0.595740795135498, -1.222450852394104, 0.1705036759376526, -2.605774164199829, 1.338616967201233, 0.35448509454727173, -0.26170170307159424, 2.700212001800537, -1.397812008857727, 0.27006545662879944, 1.1861572265625, -0.6502308249473572, -0.9803791642189026, -0.3471647799015045, -0.5723398327827454, -0.5702088475227356, -1.448667049407959, 2.7710654735565186, 0.1794401854276657, 0.8951717615127563, -0.28503531217575073, 0.7108233571052551, 0.4092843532562256, -0.08058591187000275, -1.4407292604446411, 0.34179437160491943, 0.8412719964981079, 1.027909755706787, -2.06520676612854, 0.08311709761619568, 0.6407034397125244, 0.3312705159187317, 0.2267787754535675, 1.09926176071167, 0.8301357626914978, 1.3325748443603516, -0.5405079126358032, 0.904979407787323, -0.5152260661125183, 0.21973247826099396, -0.4895334541797638, 2.243563413619995, -1.3721740245819092, 0.2927381098270416, 1.609036922454834, 0.17887544631958008, -0.3279156982898712, -0.40474840998649597, 0.25579971075057983, 0.3788977563381195, -0.04659204185009003, 0.5397241115570068, -1.3982008695602417, -0.9322419762611389, 1.0089881420135498, -0.07072248309850693, -0.27365726232528687, 0.8948601484298706, 0.34230998158454895, -0.7914880514144897, 0.4563583731651306, 0.1800042986869812, 0.3030463755130768, 0.10959155857563019, -0.2081562876701355, -0.4241795539855957, -0.7647191882133484, 0.07178223878145218, -0.08003272116184235, -0.7022271156311035, -1.956663727760315, 0.5200225114822388, -0.4853290915489197, -0.46465155482292175, 0.691432774066925, -0.7367307543754578, 0.5294610261917114, 0.07564615458250046, 0.19121053814888, -0.014881176874041557, -0.23071518540382385, 0.40569961071014404, -1.2298450469970703, 1.8548617362976074, -0.02797984704375267, -1.0555087327957153, -0.13692541420459747, 0.7058911323547363, -1.4468731880187988, -0.7362598776817322, 0.025977380573749542, -2.6473073959350586, 0.5744472146034241, -0.061658892780542374, -0.7878937125205994, -0.8519385457038879, -0.8803542256355286, -0.019026009365916252, 0.18010953068733215, -0.09837932884693146, 0.5697281956672668, 0.6165921092033386, -0.6324190497398376, 1.550666093826294, 0.7490363121032715, 0.42683497071266174, 0.7961742877960205, 0.07679175585508347, -0.30593597888946533, -1.010642170906067, 0.49674680829048157, 0.8027575016021729, -0.9175498485565186, 0.046182285994291306, 1.552729606628418, 0.7599107027053833, -0.3102363348007202, 1.8968462944030762, -0.4562617838382721, 0.0815458744764328, 0.9306552410125732, -0.33532869815826416, -0.07400917261838913, -0.3595576882362366, -0.5231126546859741, -0.10750345140695572, 0.2811613082885742, 0.8515191078186035, -1.6890472173690796, -0.8012802600860596, -0.6962264776229858, 1.5931450128555298, -0.6976523399353027, -0.6854451298713684, 0.8327200412750244, -0.4141201674938202, -0.33548250794410706, -0.4572877883911133, 1.42807137966156, -0.3725074827671051, 0.6293309926986694, -1.0372074842453003, -1.6845535039901733, -0.01617010124027729, -0.2565743625164032, 0.31969979405403137, 2.023590564727783, -1.0930612087249756, -0.9912711381912231, 0.4346255362033844, -0.20583781599998474, 0.260061651468277, -0.03822362422943115, 1.0673469305038452, -1.929919958114624, 0.8482518196105957, -0.6182344555854797, 0.7707468867301941, 0.3475954234600067, 2.322364568710327, -1.2314435243606567, -0.4341075122356415, -0.8958830833435059, 0.8148927092552185, 1.031728744506836, 1.0528448820114136, -1.5534617900848389, 0.36106187105178833, -0.10074242949485779, 0.5899817943572998, -0.6892057657241821, -0.4163775146007538, -1.1320122480392456, -0.25666704773902893, -0.661316454410553, 1.6946821212768555, -0.6137552857398987, 0.7275798320770264, 0.2355337142944336, 1.3809633255004883, 0.04583369195461273, -0.14016404747962952, 0.054347969591617584, 0.359233558177948, 0.6964830160140991, 0.4355873465538025, 0.6174837946891785, -0.79367595911026, 0.724422037601471, -0.6582679152488708, -1.4616023302078247, 1.2590086460113525, -0.4558728039264679, 1.2571966648101807, -1.7326778173446655, 0.8500989079475403, 0.7817459106445312, 0.020029742270708084, 1.7265349626541138, -0.5848844647407532, -0.05824033170938492, 2.1396937370300293, -1.1811903715133667, -0.6173928380012512, -0.7953323125839233, 0.03160000219941139, -0.6349095106124878, -0.5391369462013245, -0.3097072243690491, 1.3188440799713135, -0.6175576448440552, -0.7268558144569397, -0.06694969534873962, -0.18182063102722168, -0.685796320438385, 0.03590623661875725, 1.6653534173965454, -0.5610991716384888, -1.7243765592575073, -0.14918938279151917, 0.963094174861908, -0.33785727620124817, 0.2092341184616089, 1.388624668121338, 0.5498565435409546, -0.8102056980133057, -1.135209321975708, 0.6892885565757751, -0.13777421414852142, -1.684183955192566, -0.4553166627883911, 0.5808738470077515, 0.026180613785982132, -1.3096647262573242, 1.158116340637207, 0.1549922674894333, -0.45487332344055176, -1.0086532831192017, 0.2817896008491516, 0.3087831139564514, 0.7851797342300415, 0.12802565097808838, 1.1147522926330566, -0.2665661871433258, -1.566049337387085, 0.1440305858850479, 0.20208461582660675, -0.6344966292381287, -0.3035184144973755, 0.139312744140625, 0.5725396871566772, 0.6932547688484192, 0.178948312997818, -2.4883675575256348, 1.5848875045776367, -0.6370900273323059, 0.4896005094051361, -1.5833104848861694, 2.0831165313720703, 0.7272587418556213, 0.9896682500839233, 0.686884880065918, 0.1495089828968048, 0.9549791812896729, 0.5470250844955444, 0.0675751194357872, -1.583014726638794, 0.7004441618919373, 0.07292897999286652, 0.7556688189506531, 2.338594436645508, -1.4924663305282593, -1.1154080629348755, -0.407462477684021, -1.393570899963379, -0.5042240023612976, 0.8273309469223022, 0.916926383972168, -1.3246142864227295, 0.5362414717674255, -1.3594852685928345, -0.7067953944206238, 1.611960768699646, 0.41503462195396423, -1.5446792840957642, 0.17396028339862823, -0.10819967836141586, 1.8987014293670654, -1.2772667407989502, -0.5974779725074768, 1.463078260421753, 0.4873213469982147, -0.09165437519550323, 0.13660185039043427, 0.7817419171333313, 0.5087664127349854, -1.228440523147583, 0.9845712780952454, -1.8516829013824463, 1.1235820055007935, -0.7416420578956604, 0.5588607788085938, -1.1377860307693481, -0.7023640275001526, -0.19317986071109772, 0.8087364435195923, -0.5982218384742737, 0.5914639234542847, 0.6926661133766174, 0.9207069277763367, -0.889634907245636, -1.1274020671844482, 1.3934686183929443, 0.4642236828804016, -1.0300583839416504, -1.243377923965454, 0.6978793740272522, 0.6207650899887085, 1.8934063911437988, -0.7050851583480835, -0.9927200675010681, -1.7440953254699707, -0.14920000731945038, -1.2001569271087646, 0.8019065856933594, -0.34981536865234375, 1.6991853713989258, 0.7634080648422241, -0.6853184700012207, -1.0221152305603027, -0.5815185904502869, 0.1484137624502182, 0.44420325756073, -0.6630057096481323, 0.827312707901001, -0.756716251373291, 1.8032671213150024, 1.0781700611114502, -0.5790590047836304, -0.1739163100719452, 0.47642314434051514, 0.280734121799469, -1.147181510925293, -0.7547219395637512, -1.5937700271606445, -0.8122183084487915, -0.9939756393432617, -0.16805291175842285, 1.2663387060165405, -0.49034634232521057, 0.7136639952659607, 0.9327658414840698, 0.9049814343452454, 0.5218791961669922, -0.15120674669742584, -0.5889275670051575, -1.046534776687622, 1.0744819641113281, 0.6514521241188049, 0.7330442667007446, -1.100239634513855, -1.3421001434326172, -1.3982890844345093, 0.52046138048172, -0.39408060908317566, -1.7296892404556274, -1.5164076089859009, 0.9320486187934875, -1.630194067955017, -0.03921865299344063, -1.3799041509628296, -1.0924016237258911, 1.5654890537261963, 0.9105688333511353, -0.05603336915373802, 0.6465495824813843, -0.45571985840797424, -0.25749143958091736, 0.06927335262298584, 0.37816184759140015, 0.5632774233818054, -0.03466523811221123, -0.3565828204154968, 0.4823382496833801, 0.03808040916919708, -0.4947200119495392, 0.6735790967941284, 1.7708637714385986, -0.9240479469299316, 0.8279619812965393, -1.2177597284317017, -0.12197237461805344, -1.2505398988723755, 0.1661505252122879, 0.7083947062492371, 0.6531080603599548, -0.9431464672088623, -1.7182528972625732, -0.43051162362098694, 0.9596714973449707, 0.5606896877288818, -0.2358458787202835, 1.2223405838012695, -1.7168529033660889, 2.355422019958496, -0.300676167011261, -0.4789960980415344, 0.2900583744049072, 1.1021450757980347, 1.1078059673309326, -1.372750163078308, -1.2754261493682861, -1.0166220664978027, -0.41084206104278564, -0.5464734435081482, 1.8004804849624634, -0.6171566247940063, 1.370822548866272, -0.08710487931966782, 0.4454346299171448, 1.1068408489227295, 1.4357497692108154, -0.5843785405158997, 1.3438807725906372, -0.4582517147064209, 0.329447478055954, 0.5598844289779663, 0.9192423224449158, -0.8134185671806335, -1.7985846996307373, 0.2690526843070984, -0.49306803941726685, -0.20407113432884216, -0.1619809865951538, -0.8487797379493713, 2.2602317333221436, 1.207953929901123, -0.13196510076522827, 2.8472821712493896, -0.35610488057136536, -0.23176085948944092, -0.9728811383247375, 0.05575192719697952, -1.162065029144287, 0.8092717528343201, -1.1988736391067505, 1.0810877084732056, -1.2913410663604736, -0.4459993243217468, 1.3596723079681396, 0.9647109508514404, -2.380518913269043, 0.4703948497772217, -0.3176696300506592, 2.507413864135742, 0.46503007411956787, -0.4147797226905823, -0.7274100184440613, -0.9256003499031067, 3.3434712886810303, -0.36025071144104004, 2.25248384475708, 1.0039629936218262, 1.6412700414657593, 0.5292674899101257, 0.38165083527565, 0.11467945575714111, -0.012528138235211372, 0.00234623858705163, -0.9703661799430847, -0.31750738620758057, 1.532967448234558, -1.8174293041229248, 1.0881094932556152, 1.5570296049118042, 0.7195404767990112, 0.09240653365850449, -1.06850266456604, 1.152591586112976, -0.6553120017051697, -0.5348818302154541, -0.7649633884429932, 0.16377045214176178, -0.7733150124549866, -0.24665535986423492, 1.1083674430847168, -1.1541022062301636, -0.3091001808643341, 0.8508620262145996, 1.2586443424224854, 0.27496814727783203, 0.2103714942932129, 0.7121446132659912, 0.9687218070030212, -0.7581115961074829, -0.24935497343540192, -0.37768375873565674, -0.5024508237838745, 0.013733875006437302, 0.9592373967170715, 0.16365858912467957, 0.6991173028945923, -0.10939629375934601, -0.5234084129333496, -0.5690014362335205, -1.0436002016067505, -1.2004237174987793, 0.02406139113008976, -1.4935858249664307, -0.818080723285675, -1.2539159059524536, -2.015263319015503, -1.8642560243606567, -0.7447884678840637, -0.2581753730773926, 0.8200429677963257, 0.8970603942871094, 0.46909892559051514, 0.43581777811050415, -1.2921642065048218, -0.4068869650363922, 1.2441948652267456, 2.263576030731201, 0.6533619165420532, 0.8544022440910339, -1.7478177547454834, 0.5592261552810669, 1.957550048828125, 0.37098920345306396, 0.32584285736083984, 1.9840459823608398, -0.7468361258506775, 0.4591374397277832, 1.2901304960250854, 0.7598081231117249, 0.24256835877895355, -1.8496428728103638, -1.750546932220459, 1.53803551197052, -1.2604961395263672, 1.3534554243087769, -1.1954559087753296, 0.6495906114578247, 0.13286682963371277, 0.9172079563140869, 0.5358219742774963, -0.3083058297634125, -1.5730879306793213, -2.6780877113342285, 0.4004214406013489, 0.4755100607872009, 0.12060588598251343, 0.09246017783880234, -1.4112813472747803, -0.2081575244665146, -0.22358886897563934, -0.034943629056215286, 0.15808911621570587, -0.841448962688446, -1.4396530389785767, -0.20374980568885803, -0.8658607602119446, -1.010608434677124, -0.6751843690872192, 1.1384741067886353, -0.5542109608650208, 0.10079951584339142, 0.5665322542190552, -1.577641248703003, 0.12887045741081238, -0.4708751440048218, -0.023747604340314865, -0.0020151352509856224, -0.30449849367141724, 0.34143370389938354, 1.2895396947860718, 1.2092021703720093, -0.6879873871803284, -0.024814827367663383, 1.7356191873550415, -0.5357710123062134, -0.7552692890167236, 0.07924345135688782, -0.18007998168468475, 0.6188278794288635, -1.5699599981307983, -0.8959129452705383, -0.015237848274409771, 0.4212370216846466, -0.4477600157260895, 0.062188826501369476, 0.437334269285202, 2.3644840717315674, 0.2719847857952118, 0.8217916488647461, -0.6260242462158203, 0.9558201432228088, -1.4283668994903564, -0.05979185178875923, 0.05594127997756004, -0.5479453206062317, 1.4028409719467163, 0.2653826177120209, -1.1548914909362793, 0.610025942325592, 0.31276392936706543, -0.8531847596168518, 1.5447149276733398, 1.0218480825424194, 1.5312368869781494, 0.92954421043396, -0.18027275800704956, 0.8153901696205139, 0.6244218349456787, 1.2653074264526367, 0.31858351826667786, -0.6644607186317444, 0.8146886229515076, 0.34357285499572754, -0.5717531442642212, -1.4203424453735352, 0.03144931048154831, 0.6321541666984558, 0.03435257077217102, 0.5341607332229614, -0.4614744484424591, 2.0977823734283447, 0.4306185841560364, 0.9716458320617676, 0.05844969302415848, 2.193601131439209, 1.197364091873169, 0.4224851131439209, -0.7905207872390747, 0.7400985360145569, 0.5341318249702454, 1.8606981039047241, -0.966820240020752, 0.1060071811079979, 2.0752387046813965, 0.2819693684577942, 1.2473785877227783, 0.4231792986392975, 0.5414047837257385, 2.6661365032196045, 0.2624339163303375, 1.2535368204116821, -1.3665450811386108, -0.9787470102310181, 1.0555673837661743, 0.6033262014389038, -0.6175394654273987, -1.1731107234954834, -0.8338024616241455, 0.27530619502067566, -0.2779839336872101, -0.5891465544700623, -0.698337197303772, 0.7843254208564758, 1.0940661430358887, 0.24835483729839325, 0.5890392661094666, -0.9772985577583313, -1.5027813911437988, -0.8540111184120178, 0.039995741099119186, 1.564400553703308, 0.08789065480232239, 2.462298631668091, 0.8228502869606018, -0.3035077452659607, -0.4506959319114685, -0.6139414310455322, -0.4152548909187317, 1.619211196899414, 0.9568041563034058, 1.2484370470046997, 1.8559046983718872, 1.717301607131958, -0.9562055468559265, 2.9231016635894775, -1.8765143156051636, -1.1407272815704346, -1.1536080837249756, 1.1772648096084595, -1.0474145412445068, 1.0980464220046997, 0.631456732749939, -0.4009464681148529, -0.6979992389678955, 0.17986294627189636, -0.034680597484111786, 1.5715821981430054, -2.4050495624542236, -0.14170955121517181, -1.1459941864013672, -0.583297610282898, -0.42605361342430115, -0.23398253321647644, -1.1375868320465088, -1.036319613456726, 0.07447806745767593, -0.4809738099575043, 0.9499805569648743, 2.559404134750366, 1.2744876146316528, 0.6356781721115112, -0.22761045396327972, -1.3239662647247314, 0.33578574657440186, -1.289513111114502, -0.11195898056030273, -1.003172516822815, -0.30195778608322144, 0.03767399862408638, -1.0419204235076904, 0.9818891286849976, 0.5892415046691895, -1.5080783367156982, -0.836560845375061, -1.4055572748184204, -0.967517077922821, 0.9329987168312073, 0.30431628227233887, 0.8457531929016113, 1.4914937019348145, 1.507902979850769, -0.9884315729141235, 1.8675442934036255, 0.8250849843025208, 1.125870943069458, -1.264987826347351, -0.7787761092185974, 0.747006356716156, -0.33921319246292114, -0.5822184681892395, -0.7590010166168213, 0.08879393339157104, 0.6134829521179199, 0.6026777029037476, 2.1467435359954834, 0.4480178654193878, 1.3097413778305054, 0.8973742127418518, 0.00558155681937933, 1.1400498151779175, -1.1009660959243774, -0.3049545884132385, 0.10685395449399948, -0.6054580807685852, -1.9943100214004517, -0.08213550597429276, -0.24967873096466064, 0.21331928670406342, -1.1419402360916138, 1.0268412828445435, -3.041304349899292, 0.8227821588516235, -0.0034754136577248573, -0.15686838328838348, 0.018800506368279457, -0.5940115451812744, -1.1262259483337402, 0.8563719391822815, -0.021234918385744095, -0.3633631467819214, -0.05208136513829231, -0.6966245770454407, 0.4691278636455536, 0.7919309735298157, -0.32203102111816406, 0.24709270894527435, 1.745758056640625, -1.2025223970413208, 0.6711863279342651, -0.05468789488077164, 1.6026111841201782, -1.089857578277588, 0.46678677201271057, 0.7851249575614929, 0.561622679233551, -0.2923485040664673, -1.3143253326416016, -0.80462247133255, -1.0025572776794434, -1.3006384372711182, -1.021561861038208, 0.09651435166597366, -0.7992585897445679, -0.7100520133972168, -0.10523291677236557, -0.757277250289917, 0.41795825958251953, -0.2083899825811386, 0.8123515248298645, -0.5052034258842468, -0.7786543369293213, 0.028239531442523003, -0.33129721879959106, -1.1141356229782104, -1.778883457183838, 0.4799254834651947, -0.190511554479599, 0.20018704235553741, -0.6905288696289062, -1.0496159791946411, -1.6735076904296875, -0.3365844190120697, 0.265604168176651, -1.2745939493179321, 1.3089691400527954, 0.4394604563713074, -0.17955103516578674, -0.27521660923957825, 0.9027703404426575, 1.1460456848144531, -0.054656900465488434, -0.6502789855003357, -2.4952311515808105, -1.5659972429275513, 0.617555558681488, 0.8651894330978394, 1.5994806289672852, -0.19030441343784332, 1.5005093812942505, 1.9810521602630615, 0.2767557203769684, 0.02400786057114601, -0.3568476736545563, 0.9805665016174316, 0.6993932127952576, 0.1343212127685547, 0.267647922039032, -0.39806753396987915, 0.7829737663269043, -1.0042835474014282, -0.24743643403053284, -0.22920344769954681, 0.8210688233375549, -0.47806939482688904, -1.1607822179794312, 0.10995551198720932, -1.0515789985656738, 1.2301597595214844, 2.135704278945923, 0.4104008376598358, 0.23024095594882965, 0.21889054775238037, 0.4186477065086365, 1.5197153091430664, 0.03831403702497482, 0.15806308388710022, -0.574364185333252, -0.49134454131126404, -0.01898742839694023, -1.6028238534927368, -0.7052839994430542, -0.2663556635379791, 0.3461412191390991, -0.45315149426460266, 0.3881455063819885, 0.4498709738254547, 0.9986981749534607, 0.3956809937953949, -0.7260002493858337, -0.06892286241054535, -0.0027012138161808252, -0.04445314407348633, -1.564906358718872, 0.4217619001865387, 0.28082844614982605, -1.6765352487564087, -0.11702710390090942, -0.5987797975540161, -0.21672213077545166, -0.27765658497810364, -0.9922688603401184, -0.7561701536178589, 2.584489583969116, -0.8659266233444214, 0.41335102915763855, -0.6488955616950989, 0.8266472816467285, -1.53566312789917, -2.887479782104492, -0.07248985022306442, 0.38415250182151794, 0.052361976355314255, -1.6838458776474, 0.27210307121276855, 0.03706821799278259, -0.46450528502464294, 0.2211877405643463, 0.48498061299324036, -2.5539350509643555, -0.7159649133682251, 1.2014071941375732, 0.5721702575683594, -0.9796525835990906, 1.4306343793869019, -0.3209420144557953, -1.2586485147476196, -0.5358318090438843, -1.272711157798767, 1.1664092540740967, 0.8554638028144836, -0.6809660196304321, -0.3627684712409973, 1.538313388824463, -0.8706623911857605, -1.6080031394958496, 1.3267379999160767, -0.10513381659984589, 0.20664186775684357, 2.5948715209960938, -0.4562956690788269, -1.2685976028442383, -0.9327258467674255, -0.39308440685272217, 1.2552013397216797, -0.1137031689286232, -1.0818300247192383, 0.37904876470565796, 1.2661242485046387, -1.392706274986267, 1.7270841598510742, 0.7981560826301575, 1.3978049755096436, 0.6858354210853577, -0.6754341721534729, 0.65522301197052, -1.691373586654663, 2.021972417831421, 0.6471613645553589, -0.31976914405822754, -0.07344327867031097, 0.9388967156410217, -0.6100152134895325, -0.0670367106795311, -0.9289357662200928, -2.4704716205596924, 0.5588350296020508, -1.4053481817245483, -1.8175313472747803, -0.5835034847259521, 0.979269802570343, -0.9704271554946899, 0.7558965682983398, 0.2662416696548462], [0.7370585799217224, -0.08059144020080566, 0.0143855856731534, -0.3911247253417969, -0.46494513750076294, -0.06762894988059998, 0.40242254734039307, -0.2592327296733856, -1.680421233177185, -0.3616240918636322, -2.5546109676361084, 2.159872531890869, 0.11824372410774231, -0.41484907269477844, 0.16643860936164856, 1.7075345516204834, 0.7285947203636169, 1.0656174421310425, -0.1922595500946045, 0.30306676030158997, 0.3606102168560028, -2.4519383907318115, -1.7239973545074463, -0.12286125868558884, 0.6267532110214233, 0.8886464834213257, 0.44631531834602356, 0.16271013021469116, 2.4323363304138184, -0.18694138526916504, -0.6734205484390259, 0.3593777120113373, -0.13271814584732056, 0.2289668321609497, 3.3250601291656494, 1.7541179656982422, -1.2499780654907227, -0.30711767077445984, -0.5767387747764587, 0.7433305382728577, -0.8991454243659973, 0.7390005588531494, 0.36488592624664307, 1.3129746913909912, -0.3443339765071869, -0.005690055899322033, 0.02333875000476837, 1.028847098350525, -0.7836323380470276, -0.01792605035007, -1.486826777458191, 1.306686282157898, 0.09501495212316513, -0.6147383451461792, -1.3204984664916992, -1.9768531322479248, -0.38738250732421875, -0.5101918578147888, -1.0630896091461182, -0.13064897060394287, 0.2510433793067932, -0.6955761909484863, 1.2158384323120117, -0.34051278233528137, 0.9501394629478455, 2.055032253265381, -1.0638434886932373, -0.34485650062561035, 0.9677503705024719, -0.25452539324760437, -1.091497540473938, 0.47966453433036804, -0.20750834047794342, -0.860751211643219, 1.9059224128723145, -0.10985850542783737, 0.3154602348804474, -0.11301849037408829, -0.20844174921512604, -0.3954753279685974, 0.24947915971279144, 1.871102213859558, -1.2173435688018799, 0.04359957575798035, -0.2422773540019989, -0.9947719573974609, 0.17910747230052948, 1.452211856842041, -0.17130756378173828, -0.45508867502212524, 0.01480232272297144, -0.3064922094345093, -1.1261810064315796, -1.6767796277999878, 0.5606868267059326, 0.048954304307699203, -0.6751373410224915, -0.01299987081438303, 0.03567953407764435, 0.16147373616695404, 0.45761802792549133, 0.6702934503555298, 0.8111808896064758, 0.801820695400238, 0.22150905430316925, -0.1797369420528412, -0.4246148467063904, 1.4908970594406128, -1.4640696048736572, -0.08340374380350113, 0.23714421689510345, -0.809413731098175, 0.7787894010543823, 1.2117189168930054, 0.15472450852394104, 1.6440963745117188, 1.1435868740081787, 0.5047681331634521, 1.6016218662261963, -0.4979030191898346, 0.6158470511436462, -0.9644393920898438, -1.5990933179855347, 0.49960339069366455, -0.8843212127685547, 1.6819664239883423, 1.4205442667007446, 0.012928755953907967, -0.5350579619407654, -1.8765029907226562, -0.8157432079315186, 0.17208118736743927, 3.107726812362671, -0.7920467853546143, -0.12319665402173996, -0.6037619113922119, -0.019281810149550438, -1.0393553972244263, -1.836633324623108, -0.6151173114776611, -0.20551590621471405, -1.7084213495254517, 0.7332836985588074, 1.4231261014938354, -0.16929759085178375, 1.5565781593322754, -0.03136681765317917, 1.5386830568313599, -0.3302071988582611, 0.711557924747467, -0.28293442726135254, 1.3323463201522827, 1.2662293910980225, -0.20584355294704437, -2.5057296752929688, -0.855402410030365, 0.4336896240711212, 0.0865364745259285, 1.285743236541748, 1.599364161491394, -0.9861806631088257, -1.0572643280029297, 0.6284163594245911, -0.997057318687439, 0.2677123248577118, 0.19134078919887543, 1.0323309898376465, 1.7096896171569824, 0.7221904993057251, 2.0259203910827637, 0.550314724445343, -0.752358078956604, -0.08262449502944946, -1.1022995710372925, 1.0042238235473633, 0.11065813899040222, -1.5993869304656982, -0.5451757311820984, -0.9008399248123169, -0.6417496204376221, -0.11474853754043579, 0.3738795518875122, 0.408584326505661, 0.22984136641025543, 1.4785654544830322, -1.300204873085022, -0.8817747831344604, -0.7918386459350586, -0.7034593224525452, -0.2949204742908478, 0.468222439289093, 0.5826916694641113, 0.40395283699035645, -1.5325742959976196, -0.4093533754348755, 0.5101607441902161, 2.0382015705108643, -0.6271598935127258, 0.6609817147254944, 1.4435005187988281, 0.09403301775455475, -2.394721031188965, -2.0280818939208984, -0.5551096200942993, -0.6819602251052856, 0.2949841320514679, -0.6805810928344727, 0.4893081784248352, 0.04919013753533363, -0.15085789561271667, -0.532895565032959, -0.31563106179237366, 0.5814460515975952, 0.20115862786769867, -0.16109509766101837, 0.6944609880447388, 0.849702000617981, -0.5844008326530457, 1.4145804643630981, 0.15002234280109406, 1.268263339996338, -0.9073572158813477, 0.8658241033554077, 0.17323313653469086, -1.1780672073364258, 0.6557906270027161, 2.27595591545105, -0.47897329926490784, -0.27551954984664917, 0.6015567779541016, 0.07910507917404175, 0.7426432967185974, 0.24209535121917725, -0.1306675374507904, 0.605018138885498, 0.7046347260475159, 1.1122183799743652, 0.07249539345502853, 0.5027087330818176, -1.5671675205230713, -1.101426601409912, -1.3136191368103027, -1.344188928604126, -0.0884602963924408, -0.2260124683380127, -0.33858343958854675, 2.4516284465789795, 0.39751341938972473, -1.4758304357528687, 2.7266838550567627, 1.0972763299942017, -0.2750687301158905, 1.0579845905303955, -0.32135725021362305, 0.8568229675292969, -0.2419203370809555, 0.6373105049133301, -0.02955358475446701, 0.500063955783844, -0.2612166404724121, 0.7653011679649353, -1.2886474132537842, -1.6965140104293823, -0.8459064960479736, 0.5462213158607483, -0.022057579830288887, -0.9810523986816406, 0.04113566875457764, -1.2494381666183472, 2.0713207721710205, 0.42222368717193604, 0.026556797325611115, -1.4552277326583862, -0.9021154046058655, -1.2611027956008911, 2.0642318725585938, 1.217913269996643, 0.5594647526741028, -1.6814618110656738, 1.1890171766281128, -0.9984115958213806, -0.08790688961744308, 0.022316008806228638, -0.22821590304374695, 0.9970403909683228, -0.29999491572380066, 1.3335095643997192, 0.6895601749420166, 0.39096903800964355, 0.6926942467689514, -0.3173215091228485, -0.973514974117279, -1.541707158088684, 0.12218619138002396, 0.03373827412724495, -0.3778446912765503, 0.3145875334739685, -0.23622773587703705, 0.8914685249328613, 0.4191938638687134, -0.598984956741333, -0.22348444163799286, 0.8869422674179077, -1.108436942100525, 0.7820351719856262, -0.4175219237804413, 0.18521197140216827, 1.0907484292984009, -1.0952982902526855, 0.15872600674629211, 2.9112548828125, -0.752118706703186, 1.3610460758209229, -0.7404026985168457, 0.5569101572036743, 1.0206172466278076, -1.098857045173645, 0.10847975313663483, 0.49832627177238464, -1.2613029479980469, -0.7719871997833252, -0.22696945071220398, -0.4170098304748535, -0.3150184750556946, -1.4384446144104004, -0.5756218433380127, -0.9143586158752441, 0.4386526346206665, 1.3209409713745117, -0.7466919422149658, 1.0112286806106567, -1.5474857091903687, -0.8577980399131775, -0.26656368374824524, 0.4158121943473816, 0.6396334171295166, 0.1250762939453125, 0.1687881350517273, 1.6419826745986938, 0.4175404906272888, 0.784818172454834, 0.8696168661117554, 1.901149034500122, -0.9416170120239258, -0.39762839674949646, 0.5398362278938293, -0.41445523500442505, -2.1737122535705566, 0.7244579195976257, -0.34578070044517517, -0.4403177499771118, -1.4928641319274902, 0.06932530552148819, -1.056269884109497, 0.394428014755249, 1.1966679096221924, -1.6413782835006714, -1.6815133094787598, -0.15316982567310333, -0.627774715423584, 0.6568999290466309, -0.3032493591308594, -1.0141716003417969, -1.1430617570877075, 0.44678670167922974, 2.8358683586120605, 0.03677598014473915, -0.1285439431667328, -1.5654168128967285, -0.839023232460022, 0.9417906403541565, -0.44401222467422485, 0.17484678328037262, 0.6020558476448059, -0.5276976823806763, -0.3750387132167816, -2.8818447589874268, 1.387218713760376, 0.1656758040189743, -0.5148892998695374, 0.7550580501556396, 0.006921226624399424, -1.0615613460540771, 0.13856174051761627, 1.1610337495803833, -1.93878173828125, 0.34486135840415955, -0.9680647253990173, -0.900711715221405, -0.3589261472225189, -1.2773271799087524, 0.004824578296393156, -0.046091146767139435, -0.06605903804302216, -0.47183263301849365, 0.1394178718328476, 0.15103566646575928, 0.3561820983886719, -0.004988584667444229, 0.11276306957006454, -0.7738811373710632, -1.2332390546798706, 1.6799311637878418, -1.123875617980957, -0.7765319347381592, 1.1035268306732178, -1.006324052810669, 1.0087471008300781, 0.31762099266052246, 1.2341464757919312, -0.8612860441207886, 0.5262769460678101, 0.6757984757423401, 0.77534419298172, 0.1990073025226593, -0.36663761734962463, -0.6358815431594849, -0.6473003029823303, 0.2351042926311493, 0.23892053961753845, -0.0733189806342125, -0.03298221528530121, 1.0145643949508667, -0.2913835644721985, 1.4349443912506104, 0.8989914655685425, -0.8567671775817871, -1.944488763809204, -0.0917152538895607, -0.15416908264160156, 0.5434013605117798, -0.1304432600736618, 1.00551176071167, -0.44795092940330505, -0.8239149451255798, 0.5436502695083618, -0.4113617539405823, -0.8834244012832642, 1.5769866704940796, -1.1073323488235474, -0.254941463470459, -0.4468497633934021, 0.23477256298065186, -0.7880491018295288, 1.3495066165924072, -1.9818346500396729, 0.6631917357444763, -0.28553324937820435, 0.9588140249252319, -0.31910863518714905, -0.20466887950897217, 0.814949631690979, 0.7214257121086121, -1.0999102592468262, 1.2484151124954224, -0.5924899578094482, 1.8354510068893433, 0.9537194967269897, 0.3789156377315521, 0.5052117705345154, 0.1778957098722458, -0.9638076424598694, -0.6356808543205261, -0.10989280790090561, -0.07362924516201019, 0.7437847852706909, 0.051958635449409485, -1.434375286102295, -1.3721486330032349, 0.19918949902057648, 1.0241020917892456, 0.854279100894928, -1.0299181938171387, -0.03757037967443466, -0.6951369643211365, 1.5080281496047974, 0.32333022356033325, 0.14280378818511963, -0.5875697731971741, 0.5994328856468201, -0.24501177668571472, 1.82699716091156, 1.203961968421936, -0.9588716626167297, 0.577462375164032, -0.8215224146842957, -1.5668754577636719, 0.2523952126502991, 0.1676401048898697, 0.8254136443138123, -1.678810477256775, -1.132125735282898, -0.8332351446151733, -1.2355678081512451, 0.2569684684276581, 0.6427686214447021, 1.6540195941925049, -0.18286187946796417, -1.1309834718704224, -1.128326416015625, 0.23698790371418, 1.471571445465088, 0.7276602387428284, -0.06883788108825684, 0.9545378088951111, -1.445146083831787, -0.5525350570678711, -0.9900660514831543, -0.26546671986579895, 0.3961372375488281, -0.8264543414115906, -2.42301607131958, -0.25771525502204895, 0.4829973876476288, 1.469416618347168, -0.04011835530400276, -0.8299242854118347, -0.657436728477478, -0.2858465611934662, 0.3398120105266571, -0.6684836149215698, -1.1962952613830566, 0.8865395784378052, 0.030824970453977585, -0.5615600943565369, -1.0222728252410889, 1.2613155841827393, -1.2917602062225342, -1.0014381408691406, 1.7130507230758667, 0.26053258776664734, 0.9656006097793579, -0.9924517869949341, -0.8076432943344116, 0.39666348695755005, -0.28751590847969055, 2.590184450149536, 1.596214771270752, 1.3285350799560547, -2.9240200519561768, -3.4276299476623535, 0.8004654049873352, -0.24135005474090576, 1.1728919744491577, -0.11656522750854492, -0.27881667017936707, -1.0015270709991455, 0.6958001255989075, -1.1230289936065674, -0.6158177852630615, -0.2501307725906372, 2.2602367401123047, 0.37523216009140015, -0.44540101289749146, -0.5312137603759766, 0.14534834027290344, -0.5451674461364746, 0.2737436294555664, 1.478783369064331, -1.4134795665740967, -0.5981148481369019, 2.1365272998809814, -1.692763090133667, 2.907059907913208, -0.5701807141304016, -0.08651579171419144, -0.29838815331459045, -1.7636619806289673, -1.083188772201538, -1.2861446142196655, -1.799867868423462, 0.5571219325065613, -0.20879700779914856, 0.022363772615790367, 2.4660239219665527, -0.26785120368003845, -0.535723865032196, 0.33730414509773254, 1.3085263967514038, 0.40589439868927, -0.5206329226493835, 0.6218340992927551, -0.4816449284553528, -0.30338406562805176, 0.11550971120595932, -0.16223081946372986, 1.7701256275177002, 0.23613816499710083, 0.6428195834159851, 0.2239404320716858, -1.10137939453125, 1.590919852256775, -1.0208052396774292, 0.6935575008392334, 0.7051896452903748, -0.2794593572616577, -0.8929509520530701, -0.7058388590812683, 0.2915537357330322, -0.2988869249820709, -2.1386687755584717, -1.7731893062591553, -1.6424646377563477, -1.4169102907180786, -0.12378641217947006, -0.11727261543273926, -0.576755166053772, -1.8268946409225464, 0.8318467140197754, -0.5623821020126343, 2.4277384281158447, 1.8632771968841553, -0.22107090055942535, -0.5975211262702942, 1.6783415079116821, 0.3946274220943451, -1.7167649269104004, 0.7969144582748413, 0.1214810162782669, -0.5687045454978943, -0.010986288078129292, -0.0726483091711998, 0.6699908375740051, -1.6747380495071411, 0.458637535572052, -1.2608816623687744, 0.7399235367774963, 1.4392094612121582, -0.3240715563297272, -0.07445597648620605, 1.296383261680603, -1.7851519584655762, -0.3094748556613922, -0.013059699907898903, -1.7341485023498535, -0.23901621997356415, -0.608657717704773, -1.0811491012573242, 0.00771175604313612, 0.6840169429779053, 1.120850682258606, -0.5409008860588074, 1.3001306056976318, 1.0738816261291504, -0.3648315966129303, -1.785995602607727, 0.3084147274494171, 0.17200888693332672, 0.7582923173904419, 0.6652496457099915, 0.8041543364524841, 0.7359490394592285, -0.46423617005348206, -0.07522563636302948, 0.16124045848846436, 0.4337606430053711, 1.4816889762878418, 0.2530559301376343, -0.7523564100265503, -0.9350461363792419, -0.5472654104232788, -0.260921448469162, -2.532057046890259, -1.6269837617874146, 0.9441959261894226, -0.44184479117393494, 0.22053954005241394, -0.437601774930954, -0.4968124032020569, -0.9882537722587585, 0.21184773743152618, -0.2974299192428589, 0.7426332235336304, -0.7164829969406128, -1.1681562662124634, -1.0939213037490845, 0.7277955412864685, -1.779485821723938, 1.2700691223144531, 1.4010486602783203, -0.22083601355552673, 0.14843039214611053, -0.8716719746589661, -0.21362406015396118, 0.3936479091644287, 0.5706775784492493, -0.9151181578636169, 1.2949961423873901, 0.37089839577674866, 0.16651758551597595, -1.8711868524551392, -0.7281670570373535, 0.4406574070453644, 0.15559208393096924, -0.527563214302063, 0.24822348356246948, -1.0693780183792114, 0.31819918751716614, -0.6229131817817688, 1.4089165925979614, 2.123777151107788, 0.04620255157351494, 1.2048238515853882, 2.0017988681793213, -1.1114518642425537, 0.13321740925312042, 0.23097948729991913, -0.3273056149482727, 0.706824004650116, 0.15079998970031738, 0.5940754413604736, 0.28047722578048706, 0.5817556381225586, -0.7835142612457275, -1.340806245803833, -1.0501022338867188, 0.6865017414093018, -0.5035357475280762, -1.193661093711853, 2.1822996139526367, 1.374463438987732, -0.21229714155197144, 0.4758991003036499, -1.3103853464126587, -0.1893427073955536, -1.922036051750183, 0.5108209848403931, 0.5642085075378418, 0.8161568641662598, 1.0146291255950928, 0.968079686164856, -0.8130358457565308, -0.3397047221660614, -0.3606933355331421, -0.34572869539260864, -0.4450537860393524, 0.10607261955738068, 0.6108393669128418, -0.5820186734199524, -0.756271481513977, -0.512600839138031, -1.2111655473709106, 0.06704381853342056, -1.3994437456130981, 1.9118410348892212, -0.06428667902946472, -0.6197690367698669, 0.23124809563159943, -0.14805790781974792, 0.12062126398086548, -1.6269556283950806, -1.041038990020752, 0.39142853021621704, 1.3154298067092896, -2.2438228130340576, -1.0562036037445068, 2.04071044921875, -0.42727160453796387, -0.6814464330673218, 1.5621678829193115, -1.4895116090774536, -0.3672183156013489, -0.7886616587638855, -0.8650888204574585, 2.650148630142212, 0.1857793629169464, -0.14325648546218872, -1.262410283088684, 0.3836348056793213, -0.248857319355011, 0.6046287417411804, -1.679242491722107, 0.033745571970939636, -0.20006941258907318, 0.025283506140112877, 0.6105366349220276, -1.9109488725662231, -1.673851728439331, 0.47868412733078003, -2.952470064163208, 0.6649857759475708, 0.24790355563163757, 0.3213534355163574, -0.11803290247917175, -1.6006578207015991, -0.5391583442687988, 0.1791469305753708, -0.06905116885900497, -0.657360315322876, 2.096630573272705, -0.6663800477981567, 0.48080307245254517, 0.8105447292327881, -0.001134607009589672, 1.3897019624710083, 0.7622564435005188, -1.3612523078918457, 1.7719645500183105, 0.4398041367530823, -0.3415379822254181, -0.6646782159805298, 0.2624807357788086, 0.8133839964866638, 0.20531263947486877, 1.2428752183914185, 0.018962545320391655, -0.8288195133209229, 0.6616595387458801, 0.9117007255554199, -1.0849767923355103, 0.8559705018997192, -0.02406843565404415, 1.0921281576156616, 0.31788158416748047, 1.0016294717788696, 0.11432646214962006, 0.7899506092071533, 0.9204381704330444, 1.5300283432006836, -1.4288886785507202, -1.1265431642532349, 0.31495141983032227, 0.10081510990858078, 1.9966565370559692, 1.8656983375549316, -0.9980350136756897, 0.10484151542186737, -0.47490084171295166, 0.5049684047698975, 1.0007096529006958, -0.5420506000518799, -0.06925984472036362, -0.22304703295230865, 1.2488592863082886, 1.483040690422058, 0.43930014967918396, 0.24271723628044128, -0.317620187997818, 1.4412610530853271, -0.28763675689697266, -1.950005054473877, -1.2086611986160278, -0.6262622475624084, -0.4468451738357544, -1.012300729751587, 0.28737109899520874, 0.44755473732948303, -0.3314485549926758, 0.5944420099258423, -1.5226117372512817, 0.09396938234567642, -1.6787245273590088, 1.6555176973342896, 0.24240164458751678, -1.1193783283233643, -0.5552759170532227, -1.4641988277435303, -0.6854211091995239, 1.0043630599975586, 0.301578551530838, -0.3301270008087158, -1.1967002153396606, 0.40074217319488525, 1.428067922592163, -0.27768567204475403, 0.12270250171422958, -0.8636223077774048, 0.3658476769924164, -1.0737955570220947, 0.8638274669647217, 0.9288004040718079, -0.7146891951560974, -0.5875287055969238, -1.1304905414581299, 0.8656005859375, -1.5771312713623047, 0.3740067780017853, 0.5503873825073242, -0.15297196805477142, 0.34682804346084595, -1.624842882156372, 0.48692384362220764, -0.21156033873558044, 0.9337130188941956, -0.839034914970398, -0.9646997451782227, 0.6130580306053162, 0.13900604844093323, 1.4063469171524048, -1.2663750648498535, 1.887808918952942, 0.5390524864196777, -0.6987515687942505, 0.2935014069080353, 0.3516758978366852, -1.163645625114441, -0.9694482088088989, 1.4474399089813232, -0.7768707871437073, -0.2030486762523651, 1.1828080415725708, 0.775733232498169, 0.21850928664207458, 0.1480262577533722, -1.769161581993103, 0.15863895416259766, -1.1707462072372437, -0.6138434410095215, 0.8484327793121338, -1.3427306413650513, 0.042731206864118576, -0.2834191620349884, 0.396429181098938, -0.3028489351272583, 1.2040367126464844, -1.5359165668487549, 0.6406696438789368, 0.5926120281219482, -1.00651216506958, 0.2069578915834427, 0.08197883516550064, -0.43802645802497864, -0.5153903365135193, 2.1209635734558105, -0.5088400840759277, 1.279363751411438, -0.3086516559123993, 0.12153422087430954, 0.23217104375362396, 0.38262999057769775, 0.6427330374717712, -0.9432715177536011, 0.1414552927017212, 1.19524347782135, -0.7564148306846619, 1.3404666185379028, -0.4249318540096283, -1.0911160707473755, -0.47164231538772583, -0.9156490564346313, -1.0951340198516846, -1.0050703287124634, 0.019228367134928703, 1.4870212078094482, 1.508674144744873, -0.1298253983259201, 0.5158957839012146, 1.9186314344406128, 0.17609535157680511, -0.22883373498916626, -0.6861960887908936, -0.765197217464447, -0.08845333755016327, 1.0398344993591309, -0.4306497275829315, 0.07039424777030945, 0.6686019897460938, -0.5959669351577759, 1.6359373331069946, 0.8707573413848877, 0.6390408873558044, 1.4935163259506226, 2.0752522945404053, 0.3664247691631317, 1.1062729358673096, 0.630953311920166, -0.04949628561735153, -1.3621147871017456, -1.0169998407363892, 0.19628994166851044, -0.9213117957115173, -0.9958127737045288, -0.15525813400745392, -0.29363954067230225, -0.07654834538698196, 2.68949294090271, 1.1513999700546265, 0.06435059756040573, -0.2684056758880615, 1.3613133430480957, 0.5361668467521667, 0.9477344155311584, 0.5567829012870789, -0.9587773084640503, -2.973151922225952, 2.033193588256836, -0.25651466846466064, 0.769877016544342, 0.3870275914669037, -0.3043491840362549, -0.1503302901983261, -0.6553099155426025, 0.8999369740486145, -0.8129075169563293, 1.3499364852905273, 0.0691542699933052, 0.4456942081451416, -0.9382490515708923, -1.3670810461044312, 1.7613047361373901], [1.578635334968567, 0.5172615051269531, -0.28905221819877625, -0.1535780429840088, 0.006724509876221418, 2.8377528190612793, 0.008354807272553444, 0.5704886317253113, -0.6061753630638123, 0.24755524098873138, -0.768993616104126, -0.5894567370414734, -1.3177647590637207, -0.5222880840301514, -1.4220110177993774, 0.049772296100854874, -0.9939709305763245, -1.3212372064590454, -1.4958144426345825, -1.2160882949829102, 1.5282721519470215, -0.023768696933984756, 0.3536891043186188, -1.0486048460006714, -0.5634444355964661, 0.8227040767669678, -0.716592013835907, -1.7655125856399536, 1.466854453086853, 0.30832087993621826, 0.8806549906730652, 0.5771833658218384, -0.5410946011543274, 0.8524878621101379, -0.45720309019088745, -2.2890090942382812, 0.6228150725364685, 1.2055840492248535, -0.05307198315858841, -0.4985889494419098, 0.9313837289810181, 2.451900005340576, 0.19549985229969025, 0.5279973745346069, 1.0157002210617065, -0.5920429229736328, 0.3797908127307892, -0.3250808119773865, -1.7490752935409546, 1.1905133724212646, -2.2318308353424072, -0.4439624845981598, -0.9510850310325623, 1.348041296005249, -1.1955996751785278, -2.340012550354004, -1.1068744659423828, 1.1071504354476929, -0.798142671585083, 0.3457113206386566, 1.674962043762207, 0.055780451744794846, -0.12613984942436218, -0.044395994395017624, 0.4348593056201935, -2.6398167610168457, 0.2892918884754181, 1.4963525533676147, -0.6719527840614319, 1.8576749563217163, -1.6240214109420776, 2.1794655323028564, -0.8878088593482971, -0.013205108232796192, -0.2566836178302765, -0.7127594351768494, 0.7205953598022461, 0.11105812340974808, -0.2985022962093353, 1.4100797176361084, -1.2692689895629883, 0.07202871888875961, 0.4821135401725769, -0.19181422889232635, 0.8551062941551208, -0.3865639269351959, 0.4254577159881592, 0.2712551951408386, -0.2446572184562683, -0.4112681746482849, 0.28727713227272034, -0.7725890278816223, -0.5259912610054016, 1.28055739402771, -0.06711183488368988, 0.9849379658699036, 0.37178266048431396, 0.7638642191886902, -1.493865728378296, -0.4589337408542633, -0.48130831122398376, 0.864222526550293, 0.3367101550102234, -0.5331099033355713, 1.1370617151260376, -0.4049496054649353, -0.6532543301582336, -1.171942949295044, -0.6257140636444092, -2.2525153160095215, -0.7979467511177063, 0.38341984152793884, 0.18571041524410248, 0.26187992095947266, -1.5566596984863281, 0.8808377981185913, -0.6757953763008118, 1.9737005233764648, -0.8332393765449524, -1.4466307163238525, -2.0446276664733887, -0.4274064302444458, -2.0054516792297363, 0.24335446953773499, 0.20894576609134674, -0.280711829662323, -1.8346649408340454, 0.19722141325473785, 1.5558571815490723, 1.2049174308776855, -0.41942480206489563, 0.9050500392913818, -0.38112619519233704, 0.7133423686027527, 0.20257055759429932, 1.958729863166809, 0.21341869235038757, -0.2499941736459732, 0.42648449540138245, -0.7401673197746277, 1.1233068704605103, -1.273006558418274, 0.6855695843696594, 1.616970419883728, -0.2365991324186325, -0.6656901240348816, -0.21595677733421326, -1.7579108476638794, 0.5826454162597656, 0.09778064489364624, 1.0162383317947388, 0.07564913481473923, -0.889851987361908, -1.721138596534729, 0.36159852147102356, -0.23027800023555756, 0.4711947739124298, -0.14988426864147186, -1.3481252193450928, 0.10931193828582764, 0.524242103099823, 0.015524066984653473, -1.706836462020874, 0.6418095827102661, -0.3891323208808899, -0.46935394406318665, -1.30380380153656, -0.26045501232147217, 0.3485962152481079, -0.5623186826705933, -1.014408826828003, -1.055294394493103, 0.3446664810180664, -1.05629301071167, -0.2565155327320099, -0.06249729171395302, -1.208832025527954, 1.0194315910339355, 0.2702384293079376, -0.023810824379324913, 0.832691490650177, -0.03446396067738533, 0.07717056572437286, 0.06455352157354355, 0.7958584427833557, 1.3705393075942993, 0.8461942076683044, -1.0259220600128174, -1.5445406436920166, -1.3576297760009766, -1.9047605991363525, -1.328781247138977, 0.7215976715087891, 2.9387168884277344, 1.1922640800476074, 0.20161227881908417, 1.657836675643921, -1.4824110269546509, 0.8507936596870422, -0.9906642436981201, -0.4174126088619232, 0.8107655048370361, 0.04276283085346222, -1.1464475393295288, 0.5794917941093445, 0.8033329248428345, 0.2661373019218445, 2.0032730102539062, -1.9436719417572021, -2.241607904434204, 0.37521716952323914, -0.3953782320022583, 0.40396028757095337, 0.9862180948257446, 0.0021855186205357313, -1.0535625219345093, -0.23907919228076935, -0.2565329074859619, 0.47734951972961426, 0.06301392614841461, 1.4181442260742188, 0.24969631433486938, -1.499660849571228, 0.2068854719400406, 0.7381889820098877, 0.2299100160598755, -0.23346605896949768, -0.2700929045677185, 1.7918516397476196, 0.36090654134750366, 0.2386365532875061, -2.264092206954956, 1.0895047187805176, 1.032676100730896, 1.1340895891189575, -0.42623811960220337, -0.7265564203262329, -1.7042189836502075, -2.5197176933288574, -0.026928823441267014, -0.7295626401901245, -1.602435827255249, -1.5238059759140015, -1.116385817527771, -0.31207793951034546, -0.8870473504066467, 1.4660853147506714, -1.0543479919433594, -0.049068886786699295, -0.17059326171875, -0.3826986849308014, -0.2923290729522705, 1.6923598051071167, 0.6599352955818176, -0.8829944133758545, 0.2977524995803833, -1.2938035726547241, -0.9463376998901367, 0.3817140460014343, -1.88336181640625, 0.314797967672348, -1.5210586786270142, 0.853248655796051, -1.804401159286499, -0.37696388363838196, 0.06154009699821472, -0.8789152503013611, 0.3454974889755249, -1.8240294456481934, 1.9369256496429443, 1.1345268487930298, 0.24633629620075226, -0.2169266641139984, -0.0033349222503602505, -0.6841743588447571, 1.0200613737106323, 0.4174579679965973, -1.9560383558273315, -0.8279024362564087, 2.4837868213653564, 0.22590315341949463, -0.9615967869758606, -1.2298908233642578, 0.6604843735694885, 0.22709143161773682, 0.25230416655540466, -0.9307721257209778, -0.18006184697151184, -1.0035020112991333, -0.7247927784919739, 0.7875633239746094, -0.009778445586562157, 2.160784959793091, -0.5855413675308228, -0.17647404968738556, 1.3447476625442505, 1.9225714206695557, -0.37194469571113586, -1.1885393857955933, -0.2404138594865799, 0.7838898301124573, 1.0964492559432983, 1.6893326044082642, -0.5598757863044739, -0.8859888315200806, 0.9143788814544678, -0.15403854846954346, -1.788442611694336, -0.5061243772506714, -0.5025807619094849, 1.8611873388290405, 0.9562788009643555, -1.2961986064910889, 0.7016597390174866, 0.6096901893615723, -1.6097195148468018, -0.2752353250980377, -0.7226750254631042, 0.690866231918335, 3.0498127937316895, -0.10982150584459305, 1.5626089572906494, 0.7610514163970947, -0.7516739368438721, -0.4567192494869232, 1.412112832069397, -0.37591543793678284, 0.9582463502883911, -0.6095597147941589, -0.9933733940124512, -0.18625099956989288, 1.1162680387496948, -0.3955463469028473, 1.3168106079101562, -1.2556936740875244, -1.8480345010757446, -0.44157594442367554, 1.0053666830062866, 0.35741615295410156, 0.4939022660255432, 1.7919119596481323, -0.6283974051475525, 1.3222143650054932, -0.8153300285339355, 0.48310232162475586, -1.39517343044281, 1.8822170495986938, 0.4019058048725128, 0.6983893513679504, 1.1023553609848022, 0.7292256951332092, -0.2813410758972168, 0.6829112768173218, -0.6362713575363159, -0.3797855079174042, 0.7045124173164368, -1.5417685508728027, -0.7715063095092773, 0.035656265914440155, 1.4384881258010864, 0.6329207420349121, 0.7496027946472168, -0.8117754459381104, 0.4525175392627716, -1.257557988166809, 1.125552773475647, -0.4915676414966583, -0.037554048001766205, -0.11707129329442978, -0.7572054266929626, -0.8576181530952454, -0.09755736589431763, -0.08917830884456635, 0.954338014125824, -1.2609517574310303, 0.27649858593940735, 1.0203850269317627, 0.7238742709159851, -1.3706660270690918, -0.11166999489068985, -0.6873457431793213, -0.41577765345573425, 1.2664053440093994, 0.3905143439769745, 0.10845724493265152, -0.5768641233444214, 0.20757895708084106, -0.44716876745224, 0.9406945705413818, 2.821932792663574, 0.7029278874397278, 1.0694187879562378, 1.1568048000335693, 0.4204852283000946, 0.2616208791732788, -0.3929455280303955, 0.596930742263794, 1.6054033041000366, -0.873744785785675, -0.9779797792434692, 0.42876890301704407, 0.45284974575042725, 0.7116327881813049, 0.7760298848152161, -1.2259513139724731, -0.009340568445622921, -0.36425790190696716, 1.4355608224868774, 1.5920826196670532, -0.16347675025463104, 0.4369570016860962, 1.245761513710022, -0.869915246963501, -0.9140652418136597, -1.086499571800232, -0.14843443036079407, 2.833416223526001, -0.6972147226333618, 1.2997645139694214, 1.559980034828186, -1.0949255228042603, 0.8389906883239746, -0.8162219524383545, 1.631060004234314, 0.45674335956573486, -0.5896191000938416, 0.7097570300102234, 0.6721582412719727, 1.0800307989120483, 0.5267423391342163, 0.7641843557357788, -1.4972869157791138, -0.8849824666976929, 0.32431676983833313, -0.8663661479949951, 0.7971698045730591, 0.7133501768112183, 0.522179126739502, -1.3773486614227295, -0.3151969909667969, 1.1343234777450562, -0.6562994122505188, -0.08351994305849075, 0.4696126878261566, -0.06952652335166931, -1.286498785018921, -2.8760814666748047, -1.5623167753219604, -0.34098365902900696, -0.05703084170818329, 0.45043936371803284, 0.7636275291442871, -0.6965510249137878, 0.06550703942775726, 0.7527629733085632, 0.8297440409660339, 1.193517804145813, -0.7857999205589294, 0.3226340413093567, 0.6223483681678772, -1.0329499244689941, -1.933146357536316, 1.0158305168151855, -0.9349638819694519, -0.2795797288417816, -1.2859734296798706, -0.2744401693344116, -0.7696375250816345, 0.30994927883148193, -0.1444716900587082, 1.1813653707504272, 0.21465836465358734, -0.2134132981300354, 0.5974366068840027, -0.10522452741861343, 1.415907621383667, -1.0470943450927734, 0.8768486976623535, -0.6966108083724976, -0.2751924693584442, 1.3987033367156982, 0.03103853203356266, 0.1621754765510559, -0.5812902450561523, 1.7556630373001099, 0.5892373323440552, -0.9748266339302063, 1.4871097803115845, 0.3449150621891022, -0.3160647451877594, -1.1506903171539307, -0.11499650776386261, -0.337458997964859, -0.2402692288160324, -0.01843942329287529, -1.3086737394332886, -0.08034881949424744, -1.5292264223098755, -0.6848180890083313, 1.2909566164016724, 0.21059772372245789, -1.1297541856765747, -0.5083792209625244, 0.9941102862358093, -2.319037914276123, -0.40490543842315674, -0.2380586713552475, -0.3645806908607483, -0.06762547045946121, 0.5549679398536682, 0.11790341138839722, 1.1990668773651123, 0.020118622109293938, -1.1512224674224854, -0.15086013078689575, -0.0509384423494339, 0.44834810495376587, 0.4904542863368988, -2.0866284370422363, 0.4445764422416687, 1.3999614715576172, -2.5184402465820312, -1.0230644941329956, 0.4057564437389374, -0.24209637939929962, -0.23218603432178497, 0.34517499804496765, -0.13612055778503418, -0.6657006144523621, 0.09210482239723206, 1.096602201461792, -1.3808200359344482, -1.0320802927017212, 2.431110143661499, -0.06170014664530754, 0.04713570326566696, -0.7843238115310669, 2.2222495079040527, 0.9192696809768677, 0.13540105521678925, -0.6303296089172363, -0.1716175228357315, -0.45642098784446716, 0.28911301493644714, 1.1158891916275024, 0.43962717056274414, -0.562442421913147, 1.5327060222625732, 0.21603980660438538, 1.6177079677581787, 1.8454055786132812, -0.6689066290855408, -0.44137731194496155, 0.3350389897823334, -0.00846436619758606, 0.13400092720985413, 0.5811858773231506, -1.2496144771575928, 0.819563627243042, 1.5589599609375, 0.4405066967010498, -2.223400354385376, -0.4633715748786926, -0.22916528582572937, -0.3058392405509949, -0.008629444986581802, 1.233298659324646, 0.5404771566390991, -0.28250670433044434, 2.837977170944214, -0.9006819128990173, 0.17462454736232758, -0.5372267365455627, 1.241405963897705, 0.13147962093353271, -0.5725314617156982, -1.3521385192871094, 0.0874059721827507, -0.2896321415901184, -0.4928123652935028, -0.5890526175498962, 1.7123299837112427, 0.8591312766075134, -1.2632019519805908, 1.006912350654602, 0.7207159996032715, -1.3163537979125977, -0.3537522852420807, -0.1526191681623459, 1.125023365020752, 0.29590851068496704, -0.6078055500984192, -0.04451233893632889, 1.2024389505386353, -1.1966915130615234, -0.2681625485420227, 0.5378357172012329, 0.45378410816192627, 2.151200771331787, 1.4607559442520142, 0.35376179218292236, 0.6015051603317261, 1.3386338949203491, -0.1967145949602127, -0.25693151354789734, -0.7180506587028503, 0.1439327895641327, -1.30370032787323, -0.9228889346122742, -0.555827260017395, -1.4355696439743042, 1.4397748708724976, -0.28127455711364746, -1.3776029348373413, -0.7821853756904602, 1.1584314107894897, -1.8443211317062378, 0.4811846613883972, -1.1199417114257812, -0.31089314818382263, 1.4274030923843384, -0.4974246621131897, -0.6846231818199158, -0.8450751900672913, 0.7812429070472717, -0.5866410136222839, -0.9684877395629883, -0.3277006149291992, -0.27505236864089966, 1.030362844467163, -0.6460412740707397, -1.214412808418274, 0.4125615060329437, -1.8931689262390137, 0.3755904734134674, 0.8509675860404968, -0.14403435587882996, -1.3913716077804565, 0.8351430892944336, -0.11850942671298981, -0.006141699384897947, -0.23808744549751282, 0.522176206111908, 0.4007943570613861, 0.1691569983959198, 1.0799193382263184, 2.0385656356811523, 0.5647778511047363, -0.12277209013700485, -1.2154372930526733, -0.37353914976119995, -1.1781866550445557, -1.1515092849731445, 0.9668610095977783, 0.8500126004219055, 0.13169978559017181, -0.4389320909976959, -1.590679407119751, 1.1139484643936157, 1.7275328636169434, -0.4886763393878937, -1.1948696374893188, -0.647910475730896, 1.3273534774780273, -1.2594854831695557, 1.1544684171676636, -0.34631338715553284, -0.4315309524536133, 0.21320058405399323, -1.163784384727478, -0.9824802875518799, -1.1061931848526, 0.7716779708862305, 0.47190868854522705, -0.5389009714126587, -1.0885014533996582, 1.3610920906066895, 0.4262106120586395, 2.6371543407440186, 0.6538894772529602, -0.47705137729644775, 0.7055191397666931, -0.6320503950119019, 0.6768575310707092, 0.42403873801231384, -0.003330598585307598, -0.8725524544715881, 0.5113964080810547, 0.4002298414707184, -0.7770388126373291, -0.8438812494277954, 0.5920924544334412, -1.5332081317901611, -0.6680158376693726, 1.2838306427001953, -2.278513193130493, 0.9486759305000305, 1.4161901473999023, -0.09210314601659775, 1.9084309339523315, 3.818175792694092, -0.36271020770072937, -1.7054035663604736, 2.2103307247161865, 0.29797160625457764, 1.1627780199050903, 1.4658875465393066, -0.6146692633628845, -0.9351674914360046, -0.013432417064905167, 0.9418964385986328, -0.004460510797798634, 0.4951280653476715, 1.2439310550689697, 0.7690967321395874, 1.30802583694458, -2.1162750720977783, 0.19462384283542633, 2.0154409408569336, 0.6417952179908752, -0.7582452297210693, -0.8707838654518127, 1.1863192319869995, -1.6136211156845093, 0.1404862254858017, -0.1548488587141037, -0.6770359873771667, 0.24394318461418152, -0.779315710067749, 1.2597140073776245, 0.3803369700908661, 0.3815006613731384, -0.03456459939479828, 1.347581148147583, 0.2159801721572876, -0.42071500420570374, -2.5116631984710693, 1.4051942825317383, -0.6078661680221558, -0.3350786566734314, 0.16369256377220154, 1.9138624668121338, 0.21743158996105194, -1.8334591388702393, 0.5004594922065735, -0.9557145237922668, 0.5365001559257507, 0.14571815729141235, -0.36233943700790405, 1.2592958211898804, 0.9090408086776733, 1.1544859409332275, 0.0010766556952148676, 1.119612216949463, -0.700010359287262, -0.8620640635490417, 0.808557391166687, 0.9140306115150452, 0.21759167313575745, 0.7821832895278931, 0.9984415173530579, 1.859384298324585, 1.2124983072280884, -0.9791850447654724, -1.684874176979065, 1.5084508657455444, 1.4573500156402588, 0.2908209264278412, -0.5782980918884277, 0.44979098439216614, 0.2875022292137146, -0.17506393790245056, 0.49086564779281616, 1.1654202938079834, -1.8382149934768677, 0.04750295355916023, -0.6044162511825562, 0.7727349400520325, -0.9590772390365601, 0.32424396276474, -0.8589096665382385, 0.4982645809650421, 0.10404769331216812, 1.015643835067749, 0.0002977183321490884, -2.5645878314971924, -0.3334643244743347, -1.0639969110488892, -0.73698490858078, 1.0952489376068115, -1.6712191104888916, -1.76228666305542, 0.105118528008461, 1.7349443435668945, -0.05772646516561508, 0.10516024380922318, -1.3205410242080688, -0.6007177829742432, -1.4191218614578247, -0.031347982585430145, 0.2902936637401581, -1.7901962995529175, 1.059205412864685, -2.0315563678741455, 0.12842372059822083, -0.6240931749343872, -1.4786237478256226, 2.3421030044555664, -1.0680440664291382, -0.47668132185935974, 1.6645851135253906, 0.40976351499557495, -1.8167264461517334, -1.5385948419570923, -0.02410835213959217, -0.10190441459417343, 0.18317192792892456, 0.17957651615142822, 1.2128865718841553, 0.6162052154541016, 0.10406845062971115, 0.9090516567230225, 0.16679249703884125, -0.8106604814529419, 0.735153317451477, 0.5757245421409607, 0.7266482710838318, -1.0950379371643066, -0.14753375947475433, -1.2296712398529053, 0.6219797134399414, -1.1121859550476074, 0.38990023732185364, -0.4839164614677429, -1.4854483604431152, 0.14864085614681244, -0.3939979374408722, 0.41607654094696045, -1.0055519342422485, -0.01908975839614868, -2.3389334678649902, -2.527646780014038, -0.06897031515836716, -1.2820942401885986, 1.8346612453460693, -1.7500611543655396, 1.1050889492034912, -0.313960462808609, 1.0920416116714478, -2.436913013458252, 1.9380117654800415, -0.5714656114578247, -0.5505056977272034, -1.0565011501312256, -0.4131198823451996, -0.3581258952617645, -0.4245781898498535, 0.12347790598869324, 0.13342095911502838, 0.49187326431274414, 0.5651874542236328, -1.1054332256317139, 0.8909394145011902, 0.46117934584617615, 1.8852899074554443, -1.8723863363265991, -0.2610611617565155, -2.853172779083252, 2.1439480781555176, -0.16122958064079285, 0.17927119135856628, 0.09718182682991028, -0.2566766142845154, -0.30110326409339905, -0.8743054866790771, 0.7751107811927795, -2.602107524871826, -1.3269989490509033, 1.0649815797805786, 0.840086042881012, -1.0462385416030884, 0.8584954142570496, 0.5123063325881958, -2.1446585655212402, -0.9499323964118958, -0.06613769382238388, 0.5803629159927368, 1.682026982307434, 0.28338155150413513, -0.8935893177986145, -0.9263719916343689, -0.017816245555877686, -0.991998553276062, -1.2439595460891724, -0.05401879549026489, 0.28278985619544983, -1.0565509796142578, -1.7640775442123413, 0.5037670731544495, 1.4716306924819946, 0.8668228387832642, 2.337724208831787, -0.0653780996799469, 1.642181396484375, -1.3705809116363525, 0.33243122696876526, -0.1469767540693283, -0.41202759742736816, -0.6452663540840149, 0.08305956423282623, 1.061611294746399, -1.0804930925369263, -1.2641422748565674, 0.06215887889266014, -0.39142635464668274, -0.9361523389816284, -0.9815237522125244, 0.10969121009111404, 0.4135451018810272, 1.0169330835342407, 0.18324416875839233, 0.7825540900230408, -0.682563304901123, -0.4372726082801819, -0.14882872998714447, -1.3747837543487549, 1.7683295011520386, -0.08132724463939667, -0.4947807788848877, -1.2345670461654663, -0.9088903665542603, -0.17194367945194244, 0.2548378109931946, 2.310991048812866, -1.0122425556182861, 0.4429140090942383, -2.0124380588531494, 0.4532494843006134, -0.43179214000701904, -0.8115993738174438, -0.39360275864601135, -0.4254329204559326, 0.5849595069885254, 0.36335471272468567, 0.3132248520851135, 0.10029440373182297, 0.015947388485074043, -1.043923020362854, 0.16288898885250092, -0.2570713460445404, 1.0101810693740845, 0.20113565027713776, -0.08309353142976761, 0.37741169333457947, -0.39847713708877563, -1.3425509929656982, -0.6857481598854065, 1.1415382623672485, -0.7363259196281433, -0.2843281924724579, 0.25577685236930847, 2.4795267581939697, -1.675615668296814, -1.4111090898513794, 0.4782940447330475, -0.6765301823616028, 0.2015245258808136, -0.6851044297218323, -0.1760634183883667, -0.19856904447078705, -1.3372302055358887, 0.009043470025062561, -0.7236249446868896, -0.7106516361236572, 0.6477786302566528, -0.8318676948547363, -1.7577496767044067, 0.36333754658699036, 0.41885799169540405, -0.42907559871673584, 0.9384974837303162, -0.2590083181858063, 1.6978332996368408, 0.13328145444393158, 0.9590063095092773, 1.063935399055481, -1.5720553398132324, -1.501844048500061, 1.0049384832382202, -1.8108134269714355, -0.1133880689740181, -0.08561883121728897, -0.8084619045257568, -0.3527023196220398, 0.1710730791091919, 0.05256088450551033, 3.0576388835906982, -0.6603098511695862, 1.1160097122192383, -1.0180641412734985], [1.1253423690795898, -0.6283416152000427, 0.091053806245327, -0.19502249360084534, -0.6789650917053223, 0.45766493678092957, 0.15355587005615234, 0.4870021939277649, -0.42345598340034485, -0.42793381214141846, -0.41653090715408325, -0.3959310054779053, 1.3277004957199097, 0.9016005396842957, -1.0518165826797485, 0.6999713778495789, -0.03695399686694145, -0.2592690885066986, 1.2764610052108765, 0.4304753243923187, -0.11785035580396652, 2.0821855068206787, -1.277220606803894, -0.24143768846988678, 0.8208662867546082, -1.0043123960494995, 1.8654769659042358, 0.6414003968238831, 0.1345292031764984, -0.4288330674171448, 0.24145667254924774, -0.9903261065483093, -0.17487655580043793, 0.12770500779151917, -0.7338810563087463, 0.7623225450515747, -1.48902428150177, 0.3057056963443756, 1.442368745803833, 1.3139822483062744, -0.0656701847910881, 0.24375060200691223, -0.3371262848377228, 1.4444191455841064, -0.7625695466995239, -1.1696752309799194, 1.1227262020111084, -0.7249253988265991, 1.1830896139144897, 0.655250072479248, -0.1598290503025055, -0.5230727791786194, -0.5771331787109375, 0.27641716599464417, 0.27870258688926697, -0.17813044786453247, 0.6006900072097778, -0.5649502277374268, 0.1832202970981598, -0.05476602539420128, 2.24306583404541, -0.030179131776094437, 1.9233818054199219, -0.11333867907524109, 0.17754144966602325, 0.3213414251804352, 0.2601664960384369, -0.8357275128364563, 0.5794060230255127, -0.1709897369146347, -0.32697200775146484, -0.5404452085494995, -0.9028246402740479, 0.4244031608104706, -0.28513240814208984, -0.6386144161224365, -0.21471284329891205, 0.7930806279182434, -0.08677202463150024, -0.42060816287994385, 0.32245588302612305, -1.6562190055847168, 0.5067712068557739, -0.51996910572052, 0.36629730463027954, 1.9577887058258057, 1.0347181558609009, 0.7560901045799255, -0.3331200182437897, -0.6441741585731506, -0.45803338289260864, -0.8077399134635925, 1.3426520824432373, -1.209455966949463, -1.223546028137207, 0.5050978064537048, 0.7529075741767883, -1.058508276939392, -0.6671267747879028, 0.7011266946792603, 0.04684049263596535, -1.42310631275177, -0.16090185940265656, -1.6374865770339966, -0.7705625891685486, -0.6116631627082825, -0.02049158327281475, -0.40565025806427, -0.7574063539505005, -1.1437146663665771, 2.1444571018218994, -0.23128415644168854, -0.6624578237533569, -0.35046133399009705, -1.681511402130127, 0.04993325099349022, -0.7758941650390625, -0.18471524119377136, -0.029751472175121307, 0.015341886319220066, 0.2739339768886566, 0.7521232962608337, 0.28437694907188416, 0.40621548891067505, -0.024804705753922462, 0.5132901668548584, 0.38860636949539185, -0.1816728264093399, -1.1869659423828125, -0.36656513810157776, 0.26915907859802246, 0.3860887289047241, -0.3569888174533844, -0.17096459865570068, 0.7251277565956116, 0.6932465434074402, -1.4937846660614014, 0.5899564623832703, -0.22037941217422485, -1.8817813396453857, -2.455824375152588, 0.2812381088733673, 1.0010416507720947, 1.129144549369812, 0.5331259965896606, 0.09511423110961914, 0.8404790163040161, -0.3252541720867157, 0.3708086311817169, 1.0986335277557373, -0.5920867919921875, 1.2040364742279053, 0.36742010712623596, 0.411494642496109, 0.5107793211936951, 0.6880608797073364, -1.4417829513549805, 0.0006633663433603942, -0.46023935079574585, 0.405132532119751, 0.4237845540046692, -0.7489288449287415, 1.5681037902832031, -0.8651566505432129, -0.5626874566078186, 0.3063395023345947, -2.275787830352783, 0.8007818460464478, -0.7422981262207031, 1.4525907039642334, 1.7148178815841675, 0.8622615933418274, 0.724496603012085, -1.0958200693130493, -0.8274185061454773, -0.5051106214523315, -0.1449328064918518, -1.0166306495666504, 1.292114496231079, 0.521583080291748, -0.5664100646972656, -1.132868766784668, 0.8702975511550903, 0.3111509382724762, -1.0053256750106812, -2.538930892944336, 2.1656723022460938, -0.8060099482536316, 1.3616666793823242, 1.1523480415344238, 1.7054916620254517, -0.2944720387458801, 0.33850616216659546, -1.051272988319397, 2.6417343616485596, 0.23857977986335754, 0.7967625856399536, 1.3104519844055176, -0.8523914217948914, 0.34132710099220276, -0.866649329662323, 0.17389729619026184, -0.10408628731966019, -0.44957855343818665, 0.6597117185592651, -0.6078253984451294, -1.2541106939315796, -1.5534436702728271, -0.03594084829092026, 1.1906490325927734, 0.6789054870605469, 0.15483063459396362, -0.48511365056037903, 0.7699633836746216, 0.22153005003929138, -1.3065180778503418, -0.6829450130462646, 0.536081075668335, -0.4166729152202606, 0.25032132863998413, 1.833741545677185, -0.2982957661151886, 1.2264683246612549, -0.10234726965427399, 0.8238246440887451, 1.0523370504379272, -1.3390570878982544, -0.9449054598808289, 0.4937697649002075, -0.7111477851867676, -0.8050144910812378, -0.8555657863616943, 0.03201783820986748, 1.0080955028533936, 0.9653390049934387, -1.329884648323059, -1.239327073097229, 0.5673204064369202, 1.4193259477615356, 0.19709105789661407, 1.8162423372268677, -1.124632477760315, 0.8932505249977112, 0.4154292941093445, 0.8098368048667908, -2.360410213470459, 0.7656429409980774, -0.8344572186470032, -1.2185330390930176, -0.21597744524478912, 1.196771264076233, -0.32836663722991943, 0.5102306604385376, 0.1449420303106308, 0.746155321598053, 0.18004707992076874, 0.9954747557640076, 0.13304126262664795, -0.9842492938041687, -1.4426791667938232, 0.13728682696819305, -0.5037129521369934, -0.9292266368865967, 0.5696766972541809, -0.4241110682487488, -0.8182953596115112, -0.3907053470611572, -2.0326642990112305, 0.8805475234985352, 0.5219101905822754, -0.8830311894416809, -0.994610071182251, -0.6838671565055847, 0.09379181265830994, 0.30898118019104004, -0.43679821491241455, 0.16164830327033997, 0.8590416312217712, 0.35488906502723694, -0.6100798845291138, -1.1497265100479126, 1.0852677822113037, 0.09136655181646347, 0.19463570415973663, 0.15496332943439484, -1.7974023818969727, 0.9392367005348206, -1.3033350706100464, 0.6680089235305786, 0.8093662858009338, -1.3081291913986206, 0.6339474320411682, 0.40776026248931885, -1.6356667280197144, -1.6007987260818481, 0.2603442668914795, 0.7780705690383911, -0.6698072552680969, -2.6990957260131836, -0.7865837216377258, 0.872580349445343, -0.7030400633811951, -0.7737511992454529, 1.60299813747406, 0.7277466654777527, 0.9672888517379761, 0.4929467737674713, 0.20001770555973053, -0.0031040951143950224, 0.050033099949359894, 0.2066912204027176, -0.6236152648925781, -1.6301077604293823, -0.318988561630249, -0.8403995037078857, 0.7794976830482483, -1.4327749013900757, -1.073394775390625, -0.8260132670402527, -1.290260910987854, 0.3886719048023224, 1.141887903213501, 0.016928905621170998, -0.5146916508674622, -1.3376173973083496, -0.1753753274679184, -0.6124323606491089, -1.5945121049880981, -1.6706459522247314, 0.7780007123947144, 0.053245969116687775, 1.9736272096633911, 0.546207070350647, -2.0777487754821777, -0.2451213151216507, -0.17461253702640533, -0.5461740493774414, -0.8646180033683777, -0.5514003038406372, 0.3128788471221924, 0.5028719902038574, -1.478613018989563, 0.6422809362411499, 0.5974945425987244, -0.7562350630760193, 1.3613860607147217, -0.15566295385360718, 0.9952348470687866, 2.3112294673919678, 1.3150279521942139, -2.0186007022857666, 0.0424342080950737, -1.3090901374816895, -1.5631123781204224, -0.6731067895889282, 1.192800521850586, -0.003196872305124998, 0.9605153799057007, -0.428812712430954, 0.35779157280921936, 0.8974881172180176, -0.7778850197792053, 0.8702963590621948, 2.221435785293579, 0.04424549266695976, -0.7379714250564575, -0.8369898796081543, -0.16847850382328033, 0.08053643256425858, -0.7998078465461731, 1.2878323793411255, 1.1613824367523193, -0.5071538686752319, -0.123019739985466, -0.3281252682209015, 0.30750572681427, 0.14641913771629333, -0.08860138803720474, -0.6235833764076233, 1.0589051246643066, 1.6840612888336182, -0.9171088337898254, 0.894820511341095, 0.15951110422611237, -1.0486745834350586, 0.19654230773448944, 0.7210617661476135, 0.2036619782447815, 0.7007734775543213, 1.3252679109573364, -0.06125161424279213, -1.0598928928375244, -1.0232315063476562, 1.0227338075637817, 0.08492852747440338, -0.027578648179769516, -0.27352097630500793, 0.47355398535728455, 1.3157118558883667, -1.5066943168640137, -0.04653027281165123, 0.7793734073638916, -0.20873849093914032, 0.2642582356929779, 0.466309130191803, -0.33730557560920715, -1.3674367666244507, 0.2563607096672058, -0.5533087253570557, 0.9529104828834534, -2.580367088317871, 0.44745972752571106, 0.32097768783569336, -0.4162885248661041, 1.4379072189331055, 0.7602179646492004, -0.07256995141506195, -0.08776476979255676, -0.10123101621866226, -0.39665985107421875, 0.08689673244953156, 0.642427384853363, 0.3134649395942688, -0.292582631111145, 0.5877929329872131, 0.1683908998966217, -0.2083006501197815, -0.31150561571121216, -0.010285230353474617, -0.6678115129470825, 0.9214940071105957, -1.8016456365585327, 0.1365925520658493, 1.2668229341506958, -0.08712305873632431, 2.519055128097534, -0.03301692008972168, 1.219732642173767, -0.5761860609054565, -0.5485206246376038, -0.7545017600059509, 0.719192624092102, 2.603480815887451, -0.43250882625579834, 1.2523703575134277, 0.36128509044647217, 0.5481140613555908, 0.75645512342453, 1.135840892791748, 0.49224480986595154, -0.7857608795166016, -0.20614702999591827, -1.2615944147109985, 1.4051103591918945, -0.27805978059768677, -1.5767486095428467, -0.1796690970659256, 0.136479914188385, 0.6477475762367249, 1.0461139678955078, 0.11819043755531311, -1.679195523262024, 0.12706151604652405, -0.4608258605003357, -0.7719389796257019, -0.24352115392684937, -0.7774087190628052, 0.6321128010749817, -0.17461340129375458, 0.14230279624462128, -0.08254998177289963, -0.28074806928634644, 0.0207002405077219, 0.9701736569404602, 1.221613883972168, 1.4960790872573853, -0.04293004423379898, -0.4506867527961731, 1.2507787942886353, 0.830979585647583, 0.9903246760368347, -1.1051676273345947, -0.375125914812088, -0.515527606010437, -0.27376556396484375, 0.2820892632007599, 0.8424488306045532, 0.6971551179885864, -1.832633376121521, -1.3555543422698975, -0.4383648931980133, 1.5028780698776245, -1.3763070106506348, 1.8368216753005981, 0.8733047246932983, 0.7565663456916809, -1.611190676689148, 2.397033214569092, 1.5066503286361694, -0.42667603492736816, -0.5125632286071777, -0.3738650977611542, 0.4147445559501648, 1.3576983213424683, 0.8137016296386719, 0.6447469592094421, 0.8086296319961548, -0.9887027144432068, 0.18468166887760162, 0.7193228602409363, -0.9165789484977722, 0.5102687478065491, -0.192417174577713, 0.26975104212760925, -0.14690162241458893, -1.3432228565216064, 0.8429371118545532, -0.6957684755325317, 0.880823016166687, -1.5539333820343018, -1.8200308084487915, -0.13100144267082214, -0.023032205179333687, 1.9132553339004517, -0.14580880105495453, 1.1577329635620117, 0.5856553316116333, 1.30171537399292, 1.2557679414749146, -1.7121608257293701, 0.00023683886684011668, -0.45467281341552734, 0.38699087500572205, 0.0682302713394165, -0.5646861791610718, 0.6024578809738159, 0.30371996760368347, 1.9252088069915771, -0.9304523468017578, 0.40737468004226685, -0.47058314085006714, 1.646132230758667, 1.3298075199127197, 0.7875903844833374, 1.2852717638015747, 2.1415693759918213, 0.3099912106990814, -1.6599146127700806, 1.249572515487671, -1.2484694719314575, -0.1302345097064972, -1.234870195388794, 0.4917562007904053, 0.24922163784503937, 0.27835696935653687, 0.8670992255210876, -0.5800866484642029, -0.3302381634712219, -1.501409888267517, -2.1979236602783203, 0.771609902381897, -0.9183444380760193, 0.3681459426879883, -0.8489437699317932, -0.10141322016716003, 1.1999495029449463, -0.29974818229675293, 1.4434194564819336, 0.07277949899435043, -0.41036394238471985, 0.0074604107066988945, 0.010329140350222588, 1.8767634630203247, -0.8236798644065857, -0.5713992118835449, 0.12531208992004395, 1.3182226419448853, 0.34194090962409973, -1.1614885330200195, -0.7334617972373962, 0.5429067611694336, 0.2858477830886841, -0.9910068511962891, -0.7918798327445984, -0.1276569813489914, -0.4278227686882019, 1.203048586845398, 0.2510303258895874, 0.3381192982196808, -0.6384795308113098, 0.3175402581691742, -0.32825446128845215, -0.011304674670100212, -0.38485416769981384, 1.6328712701797485, 0.6756364107131958, -0.6779835224151611, -0.05325795337557793, -1.2710469961166382, -0.5248138308525085, 0.9613511562347412, -0.44964098930358887, -0.640466570854187, -1.1038804054260254, 0.5032966732978821, 0.5326505303382874, 0.43368133902549744, -1.1367753744125366, -1.60610032081604, 0.5746976733207703, -0.2943819463253021, -1.0590537786483765, -0.6054172515869141, -0.6121187806129456, 1.0528991222381592, -0.16785287857055664, 0.470237672328949, -1.2201846837997437, -1.7386388778686523, 0.4832080006599426, -1.7052114009857178, -1.4031438827514648, 1.4235162734985352, 2.1885361671447754, -0.49416103959083557, -0.7551481127738953, 0.9496021866798401, -0.18654875457286835, -0.6083051562309265, 0.49168211221694946, 0.13167501986026764, 0.027675846591591835, 0.6854051351547241, 0.15217669308185577, -0.6291036605834961, 1.0767890214920044, -0.8935871124267578, -0.19189095497131348, -0.2691514492034912, -0.6778218150138855, -0.516182005405426, -0.1959105134010315, 0.4985189437866211, 0.1804790496826172, 2.3115782737731934, -0.16897056996822357, -0.603585422039032, 1.9513514041900635, 1.60630464553833, -0.2676895558834076, 1.0194076299667358, 1.8181254863739014, -0.2771548926830292, 0.4609341621398926, 0.7399344444274902, -1.699013113975525, 1.029930830001831, 0.4964653253555298, 1.5443835258483887, -0.6264384984970093, -1.5144603252410889, 0.35595372319221497, -1.553572416305542, -0.6172037720680237, 1.1438814401626587, 0.24128980934619904, 0.31021541357040405, 0.5212536454200745, -0.1030861958861351, -0.2292434722185135, 0.3920240104198456, -1.3430730104446411, -0.4956567585468292, 0.3753354549407959, 1.5546507835388184, 0.08906564861536026, -0.5385567545890808, -0.978496789932251, 0.41329339146614075, -0.5098661780357361, -0.5126596093177795, 0.6512343287467957, -2.0425400733947754, -0.2948690354824066, 0.6275118589401245, -1.2189348936080933, -0.7038323283195496, 0.9503761529922485, -0.9387057423591614, 0.22558677196502686, -1.390505313873291, -0.2972820997238159, 1.6158326864242554, 0.1640956550836563, -0.9593057632446289, -0.20445439219474792, 1.0084606409072876, -0.3071247637271881, -0.619313657283783, 1.8391587734222412, -0.3516821265220642, -0.411441445350647, -0.775521457195282, 1.7322003841400146, -0.16586418449878693, -1.1982526779174805, 0.13549372553825378, -0.5225821733474731, 0.058568406850099564, -1.2500560283660889, 1.333876609802246, -0.9797513484954834, 0.14062143862247467, -2.1047298908233643, -0.2611576318740845, -1.406964659690857, -0.6028401851654053, 0.6250366568565369, -1.0542491674423218, -1.0216833353042603, 1.065671443939209, -0.6557065844535828, 1.5658094882965088, 1.212996006011963, 0.12911193072795868, -0.5910003185272217, 0.663038432598114, -0.6489027738571167, 0.6662630438804626, -1.5948141813278198, -0.5041405558586121, -0.6821818351745605, -0.3418695032596588, -0.7180176973342896, -0.07808968424797058, 0.4812767207622528, -2.188918113708496, -1.5853925943374634, 1.5749045610427856, -1.902740716934204, -0.06129249185323715, 0.17138850688934326, 0.45040491223335266, -1.204940676689148, 4.8707021051086485e-05, -0.0190042182803154, -1.264652967453003, 1.070892572402954, -0.16973209381103516, -2.181816577911377, -0.9151440858840942, 0.07562973350286484, 1.5173090696334839, -0.053646914660930634, 0.659615695476532, -1.5121986865997314, 0.8059390783309937, -0.42200997471809387, -1.3847779035568237, -1.3568837642669678, 0.16843213140964508, -0.0949687585234642, 0.9550638198852539, 0.42550572752952576, -0.3766272962093353, -0.37466123700141907, -0.14999642968177795, -0.37763118743896484, -0.5158254504203796, 1.1615997552871704, -1.1232068538665771, 1.1037429571151733, 1.287217140197754, -0.13594502210617065, 0.09493949264287949, 0.901482105255127, -0.5462535619735718, 1.325412631034851, 0.2226783037185669, 0.40389248728752136, -1.2223402261734009, 0.3514080047607422, -0.2209915965795517, -0.8994660973548889, 0.04561743512749672, -1.1874945163726807, -0.018174689263105392, -0.18673747777938843, -0.264749675989151, 0.14040574431419373, -0.5709447264671326, -0.035160522907972336, 0.23462916910648346, -0.04329892620444298, -0.3692696690559387, -1.090750813484192, -1.7032047510147095, 2.136514663696289, 0.527929961681366, -0.35287725925445557, -2.565406560897827, -0.6321181058883667, -0.6202636957168579, 1.3945010900497437, 0.9507632851600647, 0.2050701230764389, -0.35406461358070374, 1.5363644361495972, 1.230766773223877, -0.19375553727149963, -0.4903627038002014, -0.5295312404632568, 0.5479856729507446, -0.1259451061487198, -1.4149281978607178, -0.2981167733669281, 0.8824052810668945, 0.2558247447013855, -0.07823286950588226, -2.394785165786743, 1.4311261177062988, 0.09858254343271255, 0.30724290013313293, 0.8007985949516296, 0.5382064580917358, -0.10690979659557343, 1.0014714002609253, -0.5707564949989319, 0.35469114780426025, -0.7030869126319885, -1.5524786710739136, 0.09636842459440231, 0.39605915546417236, -1.4353917837142944, -1.0589025020599365, -0.5275905728340149, -0.13854198157787323, 0.6338041424751282, -1.3639956712722778, -0.19093778729438782, 0.4962471127510071, 0.19410237669944763, -0.6151231527328491, -0.32437750697135925, -1.0134400129318237, -0.9063469767570496, 1.8509166240692139, -0.6595608592033386, -1.2679095268249512, -0.9265102744102478, 0.6250320076942444, 0.4840651750564575, -1.3547680377960205, -0.3351418375968933, -1.9833687543869019, 0.33348244428634644, 0.08992650359869003, -0.667194664478302, 0.4465293288230896, -1.7725123167037964, 0.8059428930282593, -1.0764154195785522, -0.6323701739311218, -0.9772066473960876, -0.804014265537262, -0.8487676978111267, 0.3910079300403595, -0.40759262442588806, -1.2594738006591797, 0.18110834062099457, -0.09870124608278275, -0.3024890124797821, -0.1801557093858719, -0.3741947114467621, 0.3520980179309845, 1.3169702291488647, 1.0998384952545166, -0.4205057919025421, -1.3531968593597412, -0.13746744394302368, -0.9546943306922913, 0.07904642075300217, -0.7178388237953186, -0.375598281621933, -0.8287873864173889, 0.11090996116399765, -1.1140577793121338, 0.9161932468414307, -1.2829514741897583, 0.5837653279304504, 0.6348704695701599, 0.0599789135158062, -0.043724045157432556, -1.0574556589126587, -0.9346153736114502, 0.2858351469039917, 1.6803425550460815, -0.6650437116622925, 0.6893836855888367, 0.9127371907234192, -0.6959517002105713, 0.13625651597976685, -0.451231449842453, 2.2183618545532227, 0.08804001659154892, -0.179277241230011, -1.6078516244888306, -0.7100348472595215, -0.5729082822799683, -0.38535961508750916, -0.44311878085136414, -0.4888903796672821, 0.6328632235527039, -0.5562623143196106, -1.6494405269622803, -0.291252076625824, -2.5955636501312256, -0.18815486133098602, 0.6101452112197876, -0.7207827568054199, 1.3293901681900024, 0.9253624081611633, -0.7832103967666626, 1.2728468179702759, 0.17364193499088287, 0.1345885843038559, -0.3735083043575287, -0.10423362255096436, -0.3270062208175659, -0.7157096266746521, -0.8903025388717651, 0.12082822620868683, -0.16721533238887787, 1.3734890222549438, -0.7970662117004395, -0.4349973499774933, 0.5387543439865112, -0.8666158318519592, -0.06716315448284149, 1.3234319686889648, 1.3357784748077393, 0.6697839498519897, -0.7610583901405334, -0.6630332469940186, 1.1162768602371216, -1.308450698852539, -0.5157987475395203, -0.5941063761711121, -1.6935856342315674, 0.11799167841672897, -0.8337015509605408, 0.5059701204299927, 0.01598208025097847, -0.4194169342517853, -0.3649997115135193, 0.018266139551997185, -0.6747820973396301, -2.435701370239258, 0.4525472819805145, -0.31598353385925293, -0.22244837880134583, -0.42038172483444214, -1.734627604484558, -0.4446709156036377, -0.15049642324447632, 0.8111044764518738, -0.6426600217819214, 0.5865259766578674, 0.5247359871864319, -1.661856770515442, 0.5620366930961609, -1.9318119287490845, -1.4817103147506714, -0.576765239238739, 0.8727607727050781, -0.9696584343910217, -0.19723768532276154, -0.45784756541252136, 0.4671020805835724, 0.6200299859046936, -0.6755119562149048, 0.11148267239332199, -1.0673869848251343, -0.06505060195922852, -0.33255553245544434, -0.2973095178604126, 0.8937531113624573, -0.19927483797073364, -2.3786544799804688, 0.27735841274261475, -1.5876450538635254, -0.4796123802661896, -2.4308888912200928, -1.9531755447387695, -1.2203028202056885, 1.0535780191421509, -0.02384001575410366, -0.5890194773674011, -0.15920263528823853, -2.0296669006347656], [-0.2255694419145584, -1.7322118282318115, -0.6099796295166016, 1.9411746263504028, 0.7790974378585815, 0.08953641355037689, -0.8005132079124451, 0.8798929452896118, -1.3956795930862427, 0.3989715576171875, 1.293250560760498, -0.06622577458620071, 0.9215342402458191, 3.8947510719299316, 0.6638232469558716, 1.0855594873428345, 0.4483441412448883, -1.2848199605941772, 1.3339059352874756, 0.052325718104839325, 2.4664547443389893, 0.4191471040248871, 1.1274733543395996, -0.47026482224464417, -0.5561394095420837, 0.6275542378425598, -0.19607235491275787, 0.5675443410873413, 0.6347482204437256, -1.5373330116271973, -0.3698470890522003, 1.7898468971252441, 0.2682938575744629, 1.8670378923416138, 1.503719449043274, -1.3734506368637085, 0.4499194324016571, 0.4532826244831085, -0.5712937116622925, -1.59163236618042, -0.4857983887195587, 0.18849903345108032, 0.25749263167381287, 1.045735239982605, -0.40639379620552063, -0.47516152262687683, 0.5126277804374695, -0.052459120750427246, 0.534547746181488, 0.5017030239105225, 0.8046631217002869, 0.2019135057926178, 0.2630562484264374, -0.7107700705528259, -0.0466771125793457, 0.55942702293396, 0.822043240070343, 0.48656731843948364, -0.4371318817138672, 0.6944904327392578, 0.4704338014125824, 0.11329019069671631, 0.5431002378463745, -0.02410225011408329, 1.1700767278671265, -1.1994147300720215, -0.01962142065167427, -0.00907337199896574, -0.44719743728637695, 1.090129017829895, -1.0090917348861694, -0.880103588104248, -0.28448185324668884, -0.9492024779319763, -0.41440558433532715, 0.7132774591445923, 0.5934153199195862, 0.12990404665470123, 0.1664937138557434, -0.8786046504974365, -1.8214993476867676, 0.0542723722755909, -1.1032241582870483, 0.2416090965270996, -0.019391382113099098, -0.7940725088119507, 0.1165260523557663, -0.053212497383356094, 1.1153112649917603, -0.3806772232055664, 1.0741806030273438, 0.9389386177062988, -0.10614654421806335, -0.2623653709888458, 1.1575652360916138, 2.103388547897339, -0.39241522550582886, -0.4847502112388611, -1.2994180917739868, 0.10680920630693436, 1.2596700191497803, -1.0450776815414429, -1.3316011428833008, -0.22609621286392212, -0.7051240801811218, -0.05560854449868202, 0.5053719282150269, -0.26373091340065, 0.5537893772125244, -0.8809172511100769, 2.31112003326416, 1.2000188827514648, -0.5010499358177185, 0.48034489154815674, 0.09787842631340027, 0.03706684708595276, -0.20587359368801117, 0.5560054779052734, 0.6339462399482727, 0.036241281777620316, 0.10432861745357513, -1.4596443176269531, -1.4688618183135986, -0.17916330695152283, -0.1685096025466919, 0.2176014631986618, 1.0274555683135986, 0.24134372174739838, -1.1827073097229004, -1.0658321380615234, -0.8300603628158569, 0.8385171890258789, 0.6366214752197266, -0.5034611821174622, 0.7555537223815918, -0.7946024537086487, 0.7554347515106201, 0.4908409118652344, 0.6978777647018433, 0.4307062327861786, -0.1298528015613556, -1.1437880992889404, -1.2113642692565918, 0.030557487159967422, 0.0704483613371849, 0.38148438930511475, 1.0902800559997559, -1.1581774950027466, 0.490568608045578, 0.11082614213228226, 1.6781444549560547, 1.77326500415802, 0.5619459748268127, -0.12536872923374176, -0.9331201910972595, -1.3120503425598145, 0.13473565876483917, 1.872381329536438, -0.4096710979938507, 0.9487718939781189, 0.9602159857749939, 1.0033886432647705, -0.8663676977157593, -0.44444912672042847, 0.7819109559059143, -0.9783844351768494, 0.05182979628443718, 0.5326167941093445, -0.2847009003162384, 2.1347687244415283, -1.3161388635635376, 0.24815082550048828, 0.701230525970459, 0.3630238473415375, -1.3575047254562378, 0.8947655558586121, 0.14598287642002106, 0.1179022565484047, -1.76154363155365, -3.629258155822754, 0.43150272965431213, 0.8780675530433655, -1.0818442106246948, -1.130632758140564, -0.5319852232933044, -0.4359551966190338, 0.7012668251991272, 0.6728267073631287, 0.5207541584968567, -3.096696138381958, -1.6430904865264893, -0.9571670889854431, 0.3577239215373993, -0.3614168167114258, 1.7938799858093262, -1.138236165046692, 0.33186599612236023, -0.030964884907007217, 2.250257730484009, 0.19770731031894684, 0.5159918665885925, 0.6645206809043884, -0.13383233547210693, 0.09127609431743622, 0.6898574233055115, 2.051098585128784, 0.24753138422966003, 0.2808508276939392, -2.0957067012786865, 0.8321660161018372, -0.9711331725120544, -0.4580230116844177, 0.18683065474033356, -2.547105073928833, 2.188209295272827, -2.9772298336029053, 0.5052353739738464, -0.3641848564147949, 0.21155428886413574, 1.1319997310638428, -0.838808536529541, -1.1145431995391846, -0.27265506982803345, 0.3131217658519745, -2.0726635456085205, -0.11736136674880981, -0.07604145258665085, 0.3449048101902008, 0.36789241433143616, 0.5038065910339355, 0.40638497471809387, -0.5657163262367249, -0.9997144937515259, 1.4095793962478638, -0.6252074241638184, 0.15656159818172455, 0.6998900771141052, 0.1912502944469452, -0.28566285967826843, -1.3359581232070923, 1.2875840663909912, 0.7515042424201965, -0.13502588868141174, 0.9832943677902222, 0.4025346040725708, 1.3735626935958862, 1.1158030033111572, -0.1738462895154953, -1.7634786367416382, 2.0259766578674316, 0.5009103417396545, -0.9207556843757629, 1.4766405820846558, 0.1545269787311554, -1.1074960231781006, 0.3648494482040405, 1.3713269233703613, 2.5592892169952393, 0.2515753507614136, 0.009573744609951973, -1.2384490966796875, 0.8293868899345398, -1.569629430770874, 1.165738582611084, 1.2222462892532349, -1.3912173509597778, 0.14978274703025818, -0.6899198889732361, 0.5890461802482605, -1.4649641513824463, -0.14580388367176056, -1.1487659215927124, 1.1228829622268677, 0.21602770686149597, -1.9279866218566895, 0.5312024354934692, 0.018023261800408363, 1.6086052656173706, -2.1864285469055176, -1.06804621219635, 0.6956365704536438, 0.5248381495475769, -0.25835371017456055, 1.0767704248428345, -1.18975031375885, -0.10699732601642609, 0.20144233107566833, 0.7641730308532715, -0.31205183267593384, -0.21769197285175323, 1.7438685894012451, 0.25707441568374634, 0.744998574256897, -0.14981834590435028, 0.9360592365264893, -0.5783786177635193, -0.8447331190109253, 0.6006035804748535, 1.211738109588623, -1.7005053758621216, -0.11263945698738098, -0.5640031099319458, -1.2774808406829834, 0.5340853929519653, -0.18756462633609772, 0.8771249651908875, 0.5831727385520935, 2.730196475982666, -0.9084885716438293, 1.0591613054275513, 0.31338879466056824, -0.5703891515731812, 1.9971387386322021, 0.40735411643981934, -0.1321018785238266, 0.050215285271406174, 0.37285682559013367, -0.9145424365997314, -0.6437334418296814, 0.06604073196649551, -1.946822166442871, -0.2626226842403412, -0.8211739659309387, 0.5519856810569763, 0.6504243016242981, 0.8492632508277893, -0.4967687427997589, -0.8381749391555786, 2.4907824993133545, 2.452516794204712, 0.8603317141532898, 1.1529592275619507, 0.4288848340511322, -0.31166645884513855, 1.224313497543335, -0.666532039642334, 0.0872960239648819, -0.037772852927446365, 0.25430232286453247, 0.3681240677833557, -1.008091926574707, 0.31783327460289, 0.17222703993320465, -0.07765281200408936, 0.16935867071151733, 1.7047100067138672, -2.5377395153045654, -0.4191281199455261, -0.014357690699398518, 0.7675962448120117, 0.11962952464818954, -0.2868804633617401, -2.005905866622925, -0.5214062929153442, 1.0451291799545288, -2.2898924350738525, 0.877207338809967, -0.3678020238876343, -0.8940147161483765, 1.0902924537658691, 0.7606456875801086, -0.3713293671607971, -0.7084997296333313, 0.9797632694244385, 1.9321237802505493, -0.32286885380744934, -0.32575297355651855, 1.4802645444869995, 2.1136438846588135, 0.43023961782455444, 1.0443726778030396, -0.6826373338699341, -0.09730887413024902, 0.41638121008872986, -0.8151024580001831, -2.0633697509765625, 0.5151689052581787, 1.0249412059783936, 1.2253092527389526, -0.3825396001338959, -0.8503367304801941, -0.887724757194519, 0.16136164963245392, 0.06653788685798645, -1.7720052003860474, 0.35011377930641174, -2.4668521881103516, 1.9313654899597168, 0.5070515871047974, 0.03630569949746132, -0.0793173536658287, -0.7350209951400757, -1.0802671909332275, -1.2702672481536865, -1.2976078987121582, 0.9418087005615234, -0.5582008361816406, 1.2638943195343018, 0.4038221836090088, 1.3533374071121216, 0.2101602405309677, 0.7129351496696472, -0.42829644680023193, -1.5437042713165283, 1.4823819398880005, 1.0160729885101318, -0.03641674295067787, 0.036154408007860184, 0.14976896345615387, 0.42420899868011475, 1.2222286462783813, -0.1078987792134285, 0.014234217815101147, 0.29610273241996765, -0.289993017911911, -0.503191351890564, -1.4659557342529297, 1.51889169216156, -0.3298755884170532, 1.0943100452423096, -0.7337341904640198, 0.35133665800094604, 0.8355968594551086, -0.3117578327655792, 0.10331103950738907, -1.4748971462249756, -1.1132228374481201, 0.9204492568969727, 0.6118268370628357, 1.225220799446106, -0.9163643717765808, 0.07245902717113495, 0.1550740897655487, 0.3336600959300995, 1.969333529472351, 0.34151211380958557, -1.2611799240112305, 0.3926769196987152, -0.09502124786376953, -1.3851300477981567, 1.0643483400344849, 0.02485482580959797, -0.6954725384712219, -0.6192420721054077, 0.8091660141944885, 1.8016327619552612, 1.129518747329712, -0.7992802262306213, 0.8039170503616333, -0.4879661500453949, -1.531781554222107, 0.45608627796173096, -0.3855203092098236, 0.24370591342449188, 0.23223048448562622, -1.1641284227371216, -0.8083723783493042, -0.6954365968704224, 0.6731856465339661, -0.25031229853630066, -1.028525948524475, 1.0796853303909302, -0.4828285872936249, 1.1578011512756348, -0.2893967926502228, 0.20303449034690857, 0.19747821986675262, 0.06685829162597656, -0.49675729870796204, -0.6425672769546509, -0.9484432339668274, -0.31574010848999023, -1.1257774829864502, 1.300869345664978, 0.719301700592041, -0.1884566992521286, 1.7034447193145752, 0.7705270648002625, 0.19918082654476166, -2.2516582012176514, -0.7382844686508179, -0.3468594551086426, -0.7192599177360535, 1.498310923576355, -0.30318132042884827, 0.03015262447297573, -1.0680173635482788, 0.7899007797241211, -2.2088921070098877, -0.5178832411766052, -0.4521962106227875, -0.07322326302528381, 1.9801512956619263, -0.7095649242401123, -0.767460823059082, -0.2439851313829422, -0.2115451842546463, 0.35144051909446716, 0.18632517755031586, -0.17533151805400848, 0.34529033303260803, -0.33893218636512756, -1.4383066892623901, -1.482519268989563, 0.7009456157684326, 1.2124578952789307, 0.11519495397806168, 0.003842995734885335, 1.0539261102676392, -0.19908897578716278, -1.4036022424697876, 1.4888290166854858, 0.44665229320526123, -1.3592312335968018, -0.25796112418174744, -1.372245192527771, 0.6005613803863525, -0.8895357847213745, 0.43418583273887634, -1.3389630317687988, 1.7329802513122559, 0.07450199127197266, -1.7869654893875122, 0.4844887852668762, -0.29064300656318665, 0.7528735995292664, -0.17279654741287231, 1.4296964406967163, -1.2045050859451294, 1.1140390634536743, 1.467726707458496, 0.207523375749588, -0.7834553122520447, 0.6780737042427063, -0.5779068470001221, 1.1252411603927612, 1.1586782932281494, 1.2901089191436768, 0.9474480748176575, 1.6720410585403442, -0.589088499546051, 0.3894074857234955, 0.41236451268196106, 0.263814240694046, -1.6343153715133667, -1.0662108659744263, 1.3778877258300781, 1.5410739183425903, 0.30533143877983093, -0.7562986612319946, -0.4646036624908447, -0.5652210116386414, 2.1345224380493164, 1.5696526765823364, -0.14001798629760742, 0.3376155197620392, -1.464834213256836, -0.1908874362707138, 2.3161158561706543, 0.1903829425573349, -0.030229689553380013, 0.30084532499313354, 0.21399728953838348, -0.06101900339126587, 2.0097923278808594, 0.11809968203306198, 2.211779832839966, -0.9857469201087952, 2.1032843589782715, 0.7537989616394043, -0.9586586356163025, -0.8622395992279053, -0.34326767921447754, -1.3542160987854004, -0.8563635945320129, -0.17813251912593842, -1.765691876411438, 0.5134880542755127, -1.1203974485397339, -0.04838208481669426, 0.4038655161857605, -0.6499025821685791, 0.14014223217964172, 0.7252324819564819, -0.24210284650325775, 0.8131511807441711, -0.985933244228363, -0.6293664574623108, 1.4730148315429688, -0.2064424306154251, -1.9386723041534424, 0.33774250745773315, -0.8918871283531189, 0.43266239762306213, 1.1268742084503174, -1.3650155067443848, -0.6731773018836975, 1.3093299865722656, -0.17344136536121368, 0.7142011523246765, -0.17808324098587036, -0.8707358837127686, -0.012436390854418278, 0.7191213369369507, 0.47196969389915466, -1.996123194694519, 0.31692904233932495, -0.7523059844970703, 0.7520366907119751, -0.07815881818532944, 0.26052069664001465, 0.27445465326309204, -0.680361807346344, -0.05125517398118973, 0.048468563705682755, 1.7817341089248657, -0.7417618036270142, 1.086287021636963, -0.7149319052696228, 0.05085314065217972, -1.1867552995681763, 0.5562896728515625, 0.26208460330963135, 1.6222646236419678, -0.47599872946739197, -1.013307809829712, -0.1450847089290619, -1.2139942646026611, -0.8672139644622803, 0.7321414947509766, 0.10618945211172104, 0.6355161666870117, -0.710040807723999, 0.9708920121192932, -1.0972986221313477, -0.48251107335090637, -0.4543871581554413, -0.8514635562896729, 1.1161242723464966, -1.2566624879837036, 1.089316487312317, 0.7762694954872131, -0.31973978877067566, 0.016875075176358223, -0.3586727976799011, -0.5913486480712891, -0.22920392453670502, -1.1418582201004028, -0.36688435077667236, -0.4434433579444885, 0.1340114027261734, -0.5457882881164551, 0.5414448976516724, 0.06919899582862854, 2.4537365436553955, 1.8520166873931885, -0.7766470909118652, -0.1979498267173767, -0.37018030881881714, -0.708959698677063, 0.48961618542671204, 0.036181434988975525, 0.27179279923439026, 0.436365008354187, -0.38294774293899536, 0.0671931803226471, -0.38961631059646606, -1.4694936275482178, -0.7086357474327087, 0.9900455474853516, -0.6562450528144836, 1.7364856004714966, 1.6446635723114014, -0.29575562477111816, -0.09645569324493408, 0.1491634100675583, -0.883918821811676, -1.1589406728744507, 0.6875981092453003, -1.2967519760131836, -0.29265978932380676, 0.6939668655395508, 0.05414479225873947, 0.6200740337371826, -0.07069292664527893, 1.1711739301681519, -2.096553087234497, 0.10774249583482742, 0.2834668755531311, 0.5298148989677429, -0.6556863784790039, 0.8387434482574463, -1.2573959827423096, 0.06846866011619568, -0.6818385720252991, 1.49140202999115, 0.1365605741739273, -0.5078082084655762, 0.5235341191291809, 0.7338866591453552, -2.326411485671997, -1.0008617639541626, -0.30461493134498596, 0.16379334032535553, -1.2459391355514526, -1.8722013235092163, 1.2745743989944458, -0.4999643862247467, -1.717191219329834, 0.8777626752853394, 0.05330132693052292, -0.4979848861694336, -1.2400310039520264, -1.1875818967819214, 1.1753541231155396, 1.5113160610198975, -0.9883524179458618, 0.180337592959404, -0.9358401298522949, -0.67669278383255, 0.5853433609008789, -1.1023370027542114, -0.2926345765590668, -1.3668094873428345, 1.3472353219985962, -0.7235004901885986, -0.4909837543964386, -0.5365144610404968, 0.2649582624435425, -2.055755615234375, -1.0794289112091064, 0.9221311211585999, 1.6103489398956299, 0.33764809370040894, 0.05515022575855255, 1.2105984687805176, -0.3993164300918579, -0.74998539686203, 2.604970693588257, -0.5238223075866699, -1.8489338159561157, 1.0384769439697266, 2.879452705383301, -0.8019413948059082, 1.26679265499115, 0.041783396154642105, 0.8907755017280579, 2.1215827465057373, 2.2794508934020996, 1.284901738166809, 1.5038425922393799, -1.6044057607650757, -0.08056811988353729, 0.29975438117980957, -1.7338849306106567, 1.232980728149414, 0.990266740322113, 1.1349093914031982, -0.027135813608765602, -0.36561286449432373, -0.9965344071388245, 0.17891371250152588, -0.786250114440918, -0.2892095148563385, -0.7238718271255493, -0.24936792254447937, -1.4009736776351929, -0.48709389567375183, 2.0412299633026123, -0.27648603916168213, 1.005637288093567, -0.47010537981987, 0.7712163329124451, 0.49159690737724304, 0.1581955850124359, -1.4146513938903809, 0.5993651151657104, 0.5780994892120361, 0.4258963465690613, 0.7341670393943787, 0.37150219082832336, -0.05012022703886032, -0.663581132888794, 1.6440037488937378, 1.0406486988067627, 0.004749891813844442, 1.3081119060516357, 0.577144205570221, -2.0816187858581543, 1.5466891527175903, 1.575257420539856, -1.2996253967285156, 1.0621429681777954, -0.371421754360199, -1.5829130411148071, 0.21524643898010254, -0.8084354400634766, 1.4991705417633057, -0.6764938235282898, 2.3308990001678467, -0.7506880164146423, -0.5846746563911438, 0.21730440855026245, -0.09752712398767471, 0.37505194544792175, -1.1730539798736572, 0.7827728986740112, 0.4938628077507019, 0.6239229440689087, 1.3087495565414429, -0.16625764966011047, -0.08938252925872803, -1.1107304096221924, -1.2401100397109985, -0.06715469807386398, -0.6652594804763794, -1.108608603477478, -0.1607600748538971, -2.0086891651153564, 0.015278127044439316, 0.15691961348056793, 1.942014217376709, -0.4387947916984558, -0.4155944883823395, -0.1508626490831375, -1.1594433784484863, 0.3558579981327057, -0.39181363582611084, -0.764531672000885, 2.094435691833496, -0.06537102162837982, -0.06730342656373978, -0.21210679411888123, -0.7957877516746521, 0.4942571222782135, 0.31303685903549194, -1.7613213062286377, 0.36042916774749756, 1.2982839345932007, -0.9788447022438049, 0.672719419002533, 0.8696652054786682, 1.1732479333877563, 0.8882266879081726, 1.4402530193328857, 0.20410645008087158, 0.44167354702949524, -1.0053882598876953, 0.029469411820173264, 1.374400019645691, -1.3971295356750488, -0.3018321990966797, -0.3015446960926056, 1.2213789224624634, -0.16939133405685425, -0.8339543342590332, 0.22866284847259521, -1.0688530206680298, 0.5044357776641846, 0.17124363780021667, -1.1279584169387817, 0.29243937134742737, 0.783223569393158, -0.8844744563102722, -1.194827675819397, 1.6885433197021484, -0.08907382190227509, 1.3473567962646484, 1.2169362306594849, 0.14620889723300934, 0.3358440399169922, 0.14398854970932007, -1.1378469467163086, -1.392740249633789, 0.5719523429870605, -1.53056001663208, 0.08860065788030624, -0.412578284740448, 0.48990756273269653, -0.5576788187026978, 0.2510119676589966, -0.8276716470718384, -0.19746385514736176, -1.8613954782485962, 1.3138574361801147, 0.784525454044342, 0.003947408404201269, 1.1835312843322754, -0.24261313676834106, -0.05015943944454193, -2.428116798400879, -1.0034148693084717, -0.10933101922273636, -1.0742263793945312, -0.8818836808204651, 1.5563987493515015, 0.8718290328979492, -1.1316893100738525, 0.005463855341076851, 0.09648183733224869, -1.1615028381347656, -2.862867832183838, 0.5346265435218811, -0.03322971239686012, -0.3163081109523773, 0.9420554041862488, 0.07289234548807144, -0.27477937936782837, 0.05558701232075691, 0.7920622229576111, -0.21843090653419495, 0.7217531204223633, -0.25998786091804504, 1.037091851234436, -1.7017927169799805, -0.19034115970134735, -0.04884345456957817, 1.3249329328536987, 0.4187254309654236, -1.8656392097473145, 0.8574029803276062, -0.6474536061286926, -0.13062989711761475, -0.5702369809150696, 0.7046213746070862, -0.7026412487030029, 0.8133348226547241, -1.527184247970581, -0.7249647378921509, 1.112671136856079, -0.9759721159934998, 1.027786374092102, 1.3706690073013306, -2.0467920303344727, 0.7008162140846252, 0.3520628809928894, 1.3268351554870605, -0.19709785282611847, 0.9150558710098267, 0.24362175166606903, -0.6452914476394653, 0.24300774931907654, 1.1739072799682617, 0.4318962097167969, 0.3976214826107025, 2.1934916973114014, -0.4625507891178131, -0.21350687742233276, 1.5875622034072876, -0.12296845018863678, 0.020301438868045807, 0.08139481395483017, 0.5212077498435974, 0.7243682146072388, 0.23913414776325226, -1.715575933456421, -0.5746215581893921, 0.09453770518302917, -1.2698084115982056, -0.22809791564941406, 0.8894363045692444, 0.7172605395317078, -0.3910525441169739, 0.6113847494125366, 0.8821619153022766, 1.1465224027633667, 0.821229875087738, -0.5243366360664368, -0.9603694081306458, -1.5657240152359009, -1.676927924156189, 0.7978973984718323, -0.2517630457878113, -0.1525861918926239, -1.2435451745986938, 1.835714340209961, -1.0937249660491943, -0.914937436580658, -1.7041053771972656, 0.344777911901474, -0.2831753194332123, 1.1003693342208862, -0.058992717415094376, -0.9475695490837097, 0.10502629727125168, 1.0350673198699951, 0.119473397731781, -1.5069091320037842, 0.3886968195438385, -0.4299852252006531, 0.25519686937332153, -0.7819617986679077, 0.38955986499786377, -1.0656901597976685, -0.40210068225860596, -0.3716990351676941, -0.5376731753349304, 0.2977067828178406, 2.6217234134674072, 0.8039020299911499], [0.7214305400848389, -1.327223539352417, -0.4982246458530426, -1.0979392528533936, 0.8815334439277649, 2.4781875610351562, 1.3392733335494995, 1.0892211198806763, -0.9576124548912048, -0.468484103679657, -1.6261940002441406, -0.7970814108848572, -1.0181750059127808, -1.1736291646957397, -0.7617676854133606, 0.21570651233196259, -0.590701699256897, -0.5569517016410828, -2.053136110305786, 0.09478078782558441, 0.34738099575042725, 0.6285642385482788, 0.6141191124916077, 0.18237069249153137, 2.338773488998413, -0.4657491147518158, -0.9813152551651001, 0.3043140470981598, 0.7573550343513489, -1.5707309246063232, -1.2602459192276, -0.19159191846847534, 0.008346086367964745, 2.332827091217041, -0.802648663520813, 0.3264240324497223, -0.5471076965332031, -0.17426355183124542, -1.017293095588684, -0.9558737277984619, -0.25632554292678833, 0.06376489996910095, 1.7417800426483154, 0.05682387948036194, 0.5458346009254456, 0.13300172984600067, -0.15649139881134033, -0.9732657074928284, 1.0311447381973267, -0.39582452178001404, 1.6640117168426514, -0.5551422238349915, -1.1913810968399048, 0.4552345275878906, -0.15465612709522247, 0.045523062348365784, -0.5005438327789307, -0.07404547929763794, -1.5745383501052856, 0.4148149788379669, 0.9649721384048462, -0.6419563293457031, -1.5584485530853271, -0.3969953656196594, -0.37603792548179626, -0.2567322552204132, 0.13169847428798676, 0.8629998564720154, -0.06232091411948204, 0.8501471877098083, 0.7788568139076233, -0.3506057560443878, -0.28286057710647583, -1.6859562397003174, -1.1050364971160889, -0.8051193356513977, 1.564483642578125, -0.9239394664764404, -0.1617061197757721, -0.6920360326766968, 0.9515920281410217, -1.4437693357467651, -0.5120442509651184, 0.938127338886261, 0.059310223907232285, 2.0257444381713867, -0.6287832856178284, -2.400528907775879, -0.9923806190490723, 0.31160464882850647, 0.3920089900493622, 0.018117135390639305, 0.2979592978954315, 1.2900363206863403, -1.3264203071594238, -0.9824737906455994, -0.9940371513366699, 1.7158105373382568, 0.5356016159057617, -0.4560166895389557, 1.0690003633499146, -1.1699492931365967, 0.17973776161670685, -1.863540530204773, -0.7054731249809265, 0.4057038128376007, 0.4878576695919037, 0.7544034123420715, -0.4272034764289856, 0.5652198791503906, 0.9945446848869324, 1.6453707218170166, -0.2968844473361969, 0.46875372529029846, -1.3666914701461792, 0.6826493740081787, 1.1232670545578003, 0.6640174388885498, 1.4089572429656982, -0.6793336272239685, 0.27148938179016113, 0.8013913631439209, -0.5541372299194336, 0.7079623937606812, -1.8795535564422607, -1.56508469581604, 0.4951975643634796, -0.5148199796676636, -0.34122636914253235, 0.22728902101516724, -0.3374083340167999, 0.6380822062492371, 1.2133030891418457, 0.4100487530231476, -1.5766289234161377, 0.6378423571586609, -1.8128756284713745, -1.1300665140151978, 1.5158439874649048, 0.3335033357143402, -0.31937339901924133, 0.8308054208755493, -0.12389209866523743, 0.6671229600906372, -1.788170337677002, 0.9240574836730957, -1.569994330406189, 1.6042890548706055, 1.2345342636108398, 1.0444144010543823, -1.4742779731750488, 0.65353924036026, -1.4734997749328613, 0.570338249206543, -0.2334711104631424, 0.8632473349571228, 1.771009087562561, 0.10883209854364395, 0.6595270037651062, -1.0186562538146973, 0.10768095403909683, 0.35527151823043823, -0.5578107833862305, 0.4235478341579437, -1.1262226104736328, -0.8583501577377319, 2.175340414047241, 0.14075234532356262, 0.10051555931568146, 0.384450227022171, -1.8840854167938232, -1.6241631507873535, 0.9456511735916138, 1.3951150178909302, 0.20524223148822784, 2.103243112564087, 0.03627778962254524, 0.9909123778343201, -0.9328184127807617, -0.12176604568958282, 0.8318547010421753, -0.5263632535934448, -0.8894359469413757, 0.2532278299331665, 0.2741204798221588, -0.7908159494400024, -1.7537492513656616, 0.7649413347244263, -1.5747225284576416, -1.9529939889907837, 2.6963582038879395, 1.1428171396255493, -0.6182432770729065, -0.21297436952590942, 0.760179340839386, -0.0844452977180481, -0.41255080699920654, 0.45680978894233704, -0.32005658745765686, -0.9649630784988403, -0.934590220451355, -1.0302306413650513, 0.14804670214653015, 0.6288118958473206, 0.1510835736989975, -1.037786841392517, -1.8288342952728271, -1.5790090560913086, 0.12280523777008057, -0.031806737184524536, 1.0805257558822632, -0.5401343703269958, -0.6211014986038208, 0.7963343262672424, 1.4480044841766357, -0.07312508672475815, -0.8471952080726624, 2.4830732345581055, 0.16286444664001465, -0.4837590157985687, 0.6363804340362549, 0.4195614457130432, 0.05716779828071594, 0.006472993642091751, -0.3862116038799286, 1.4084715843200684, -0.022712919861078262, -3.2589476108551025, -0.29934507608413696, -1.0256307125091553, -0.08569779992103577, 0.052219945937395096, -0.2507750689983368, 0.7173371911048889, 1.060030221939087, 0.10307218134403229, -0.12744224071502686, 1.4252923727035522, 0.26378852128982544, 1.6086424589157104, -2.0567269325256348, 0.7290652990341187, -0.6092175245285034, -0.021881533786654472, 0.9753017425537109, 0.9368959069252014, -1.305449366569519, -0.11493759602308273, 1.529416561126709, 0.042510345578193665, 0.25197821855545044, 1.4966161251068115, 0.27087053656578064, -0.6265267133712769, -0.27630314230918884, -1.184395432472229, 3.6275501251220703, 1.8061577081680298, 0.9785467386245728, 0.1771395355463028, -3.0901830196380615, -0.12208622694015503, 1.0981580018997192, 0.19796834886074066, 0.11028703302145004, 0.8786445260047913, -0.8166728615760803, -0.3949267566204071, -0.27170857787132263, 1.4008005857467651, -0.11109479516744614, 0.759848952293396, 0.21928095817565918, 0.8751353621482849, 0.20753400027751923, -0.11756015568971634, -0.5945625901222229, 1.326158881187439, -1.4700133800506592, -1.2206987142562866, 0.2891587018966675, 0.31302228569984436, -1.178076982498169, -0.4072026312351227, -0.17966265976428986, -0.43747836351394653, -1.230783224105835, 0.6106781363487244, -0.16036026179790497, 1.6681430339813232, 1.4100652933120728, 0.7091227173805237, -0.5851952433586121, -0.14897999167442322, -0.9247825145721436, -1.4860658645629883, 0.02311124838888645, -0.5130720138549805, 1.2635823488235474, -1.1450170278549194, -1.0370891094207764, 2.84944224357605, -1.9790788888931274, -0.084991954267025, -0.26536881923675537, -0.6049001216888428, 0.8356970548629761, 0.18106123805046082, 2.8764171600341797, 0.6790381073951721, 1.901969075202942, -1.2108491659164429, 0.6736418604850769, -0.2726448178291321, 0.4508388042449951, 0.018862321972846985, 0.025302134454250336, -0.1655808985233307, -0.5668801665306091, 0.06014016643166542, 1.8285789489746094, -1.7406857013702393, -0.9564709067344666, -0.569140613079071, 0.6199352741241455, 0.45382967591285706, -1.050439715385437, 1.5101304054260254, -0.25778141617774963, -0.47307026386260986, 0.978691041469574, 0.9096689820289612, -2.536046266555786, -0.9324464797973633, 0.41255512833595276, 1.5422219038009644, -1.8123910427093506, 0.30161574482917786, -0.38608235120773315, -0.6974183320999146, 0.20616717636585236, 0.4771304726600647, -1.4553273916244507, -1.1409730911254883, -0.9347898364067078, 0.3326363265514374, -0.6734811663627625, 0.15834900736808777, 0.2393820732831955, -0.21364644169807434, 1.5805126428604126, 0.3782632350921631, 0.044053927063941956, 0.13493528962135315, -0.22818762063980103, 0.7247470021247864, 0.1693432629108429, -1.0914413928985596, -0.9162303805351257, -0.050496120005846024, -0.00648609921336174, 0.8832139372825623, 0.21112625300884247, -0.37680038809776306, 1.3592149019241333, 1.7988989353179932, -0.26810014247894287, -1.3212614059448242, -0.7927382588386536, 0.8049821853637695, 1.619895339012146, -0.4538927674293518, 0.6392829418182373, -0.4774358868598938, 0.4393359124660492, -0.2109169214963913, -0.5376250147819519, -0.8352881073951721, 0.9034553170204163, 0.46474191546440125, 0.5623422861099243, 1.0367006063461304, 0.0298526082187891, 0.002008062321692705, -0.46424809098243713, 0.5149343013763428, 0.9516754746437073, -0.09406907856464386, -0.6312841773033142, -0.7351751923561096, -0.6936865448951721, 1.461896538734436, -0.285552978515625, -0.7004497647285461, 2.4638094902038574, -0.9066042900085449, -0.6957716941833496, 1.4831345081329346, 0.749289870262146, -1.022141933441162, 1.4442931413650513, 2.2705142498016357, 0.09789343178272247, 0.8876031041145325, 0.26729637384414673, 0.3366243243217468, -0.01665090024471283, -0.47829321026802063, -1.5531615018844604, 0.30852481722831726, 1.4887782335281372, -2.5420799255371094, 0.014405157417058945, -1.805358648300171, 0.12528476119041443, -1.5158276557922363, -0.8930360078811646, -0.48639899492263794, -1.356650710105896, -0.24219034612178802, -0.5552197098731995, 0.7641322612762451, 1.0245709419250488, 1.2073343992233276, 0.309978187084198, 0.36344802379608154, 0.9490001201629639, 1.677786946296692, 0.02268097922205925, -0.8359565734863281, 1.6076287031173706, 0.2384532392024994, 1.0683366060256958, 0.4254607558250427, -0.4365794062614441, -0.5591135621070862, -0.38443484902381897, 0.8957608938217163, -1.2705814838409424, 1.2210623025894165, 0.05685006454586983, 1.1092807054519653, -0.8831892013549805, -0.07142698019742966, -0.012723629362881184, 1.3579009771347046, -1.265939474105835, 1.1549880504608154, -0.46806085109710693, -0.7999368906021118, -0.4601369798183441, -0.4337621331214905, 1.5747641324996948, 0.5865052938461304, -1.1066032648086548, -0.8348065614700317, 0.6077003479003906, -0.38351762294769287, 0.9121412634849548, -0.5530843138694763, -1.6321215629577637, -0.5014442801475525, 0.2847525179386139, 0.008630504831671715, 0.04807494953274727, 0.11304748058319092, -0.13202539086341858, 0.5854834914207458, -0.701580822467804, 0.7001774907112122, -0.07303830236196518, -0.2304667979478836, -0.29377448558807373, -1.076815128326416, -0.6022697687149048, 0.5387511253356934, 1.5275874137878418, -1.9304585456848145, 0.4025667607784271, 0.7566586136817932, 1.3117258548736572, -1.0616189241409302, 1.2717444896697998, -0.28616514801979065, 0.04992780089378357, -0.6051151752471924, -0.0277742650359869, 0.10083527117967606, 0.3647873103618622, -0.22794519364833832, -0.45767542719841003, -0.27697333693504333, 1.0271035432815552, -0.2659189701080322, 0.012715302407741547, 2.407665967941284, 0.7306926846504211, 1.6591980457305908, 1.9484509229660034, -0.11017865687608719, 0.6690950393676758, 0.6250657439231873, -0.23153577744960785, -0.0725279301404953, 0.7862905859947205, -0.22839896380901337, 0.16177265346050262, -0.6884737014770508, 0.5060250163078308, 1.5235179662704468, 0.9606508016586304, 0.23998652398586273, 0.3735154867172241, -1.5188579559326172, 0.7558372616767883, 0.44434240460395813, 0.6254954934120178, -0.014293968677520752, 0.6061429381370544, -0.5114259719848633, 1.5563219785690308, 0.16049695014953613, -0.6314785480499268, 0.564877450466156, -3.251391887664795, 0.4536519944667816, 2.1358284950256348, 0.9711653590202332, -0.35220420360565186, 0.17155902087688446, 1.7252781391143799, 0.2325514554977417, 0.79243004322052, -0.9084246754646301, 1.3735445737838745, -1.2275354862213135, 0.8046613931655884, 0.022936731576919556, -0.08654080331325531, 1.4434893131256104, -1.6802524328231812, -0.3054322898387909, -0.09802619367837906, -1.255126953125, 1.600924015045166, -1.0267443656921387, -0.9933062195777893, 0.04930383712053299, 0.20252032577991486, 0.753237783908844, 0.3349950611591339, 0.5103099942207336, -1.2283045053482056, 0.24026243388652802, 0.5381581783294678, 1.4841262102127075, 0.5674092769622803, 0.6042594313621521, -0.6225710511207581, -0.13178172707557678, -0.3980165123939514, 0.35099735856056213, -0.9391394257545471, 0.35284313559532166, 0.22414857149124146, 1.6055887937545776, -1.2146639823913574, -1.0783076286315918, 0.6737262606620789, -0.8197326064109802, 0.7296794056892395, -1.8062101602554321, -1.337131142616272, -1.7720824480056763, -0.27188023924827576, 1.358095407485962, -1.5347577333450317, 0.7355602383613586, -1.208448052406311, -0.2333674132823944, -0.830427885055542, 0.4251408874988556, -2.0338823795318604, 2.495755195617676, 0.20693933963775635, 2.4027657508850098, 1.8975366353988647, -0.6572712659835815, 0.8591569066047668, -1.2593531608581543, 0.5579554438591003, 1.5371112823486328, 0.3035376965999603, -1.0611387491226196, 0.365467369556427, -1.0632699728012085, -0.7376583218574524, 0.3422897756099701, -0.7062124609947205, 0.3673497438430786, 0.5268774628639221, 0.23857666552066803, 0.9601544141769409, 0.6474320888519287, -0.417037695646286, 1.8720180988311768, 0.8096222281455994, -0.07126236706972122, 0.29862180352211, -1.081558108329773, -0.82023024559021, 0.5876938104629517, -1.7278696298599243, -1.9479364156723022, 0.16510134935379028, -1.059118390083313, -0.01923106424510479, -0.6438398957252502, 0.9262778759002686, 0.16511736810207367, 1.3479528427124023, 0.32457539439201355, 1.1847710609436035, 1.451032042503357, 1.9722306728363037, -0.9187341928482056, -0.27607882022857666, 1.745148777961731, 0.7107602953910828, 0.3352380692958832, 0.8452681303024292, -0.7042903304100037, 2.3102169036865234, 0.12213359028100967, 1.577235221862793, -1.2455098628997803, -0.19974671304225922, 1.236022710800171, 0.2142563760280609, -0.1132279485464096, 1.3341519832611084, -0.10409244894981384, -0.8633210062980652, -0.1750313639640808, 0.3474763035774231, 0.010058460757136345, 1.141089916229248, -1.0028131008148193, 0.7029857635498047, -0.7890875935554504, 0.7208655476570129, -0.536790668964386, 0.5268129110336304, -0.8395102620124817, -0.0366651825606823, 0.4881289005279541, -0.5358692407608032, 2.0402231216430664, 1.6467862129211426, 0.154478520154953, 1.4611119031906128, 2.129211902618408, -0.26564210653305054, -0.12196885049343109, 0.9940349459648132, -0.9938041567802429, 0.31094101071357727, -0.9899638891220093, -1.0248994827270508, 1.9387224912643433, 0.7790466547012329, 1.1435669660568237, -0.1710282862186432, -1.2536184787750244, 0.33504146337509155, 0.47336387634277344, 1.2177327871322632, 0.9283552169799805, 0.9585338830947876, -0.9425897598266602, -1.265325903892517, -1.6023262739181519, -0.44101038575172424, 1.3404661417007446, 2.4853570461273193, -1.2619929313659668, 1.2322663068771362, -0.17715738713741302, 0.8738861083984375, -1.2422250509262085, -0.36906880140304565, 0.7789957523345947, 1.7707135677337646, -0.4594302177429199, -0.03341050446033478, -1.1404963731765747, 0.6711328625679016, 0.4659689664840698, 1.7513070106506348, -0.32201114296913147, -0.9385968446731567, -1.1566201448440552, 1.8890353441238403, 1.1092545986175537, 0.38619524240493774, 1.1714175939559937, -0.4703716039657593, 1.1978532075881958, 1.0469121932983398, 0.9973151087760925, -0.9584895372390747, -1.6209568977355957, 0.6085612177848816, -0.06652163714170456, -1.4511315822601318, 0.3433223068714142, 0.09356524050235748, 0.3315731883049011, -0.6871623992919922, 1.4288830757141113, -0.5026282668113708, 0.21955899894237518, -1.4396460056304932, -1.7317612171173096, -1.93580162525177, 0.3935920298099518, 0.2554294466972351, 0.8215645551681519, -0.10639843344688416, 0.8434629440307617, -0.5581797361373901, 1.2045456171035767, -1.606465458869934, 0.31802958250045776, 0.10101359337568283, -0.26423200964927673, -1.9785592555999756, 0.13352103531360626, -0.18643803894519806, -0.4383029043674469, 0.44353747367858887, -0.4972892105579376, -1.1846961975097656, 0.787163496017456, -1.095846176147461, 1.3637213706970215, -0.8201666474342346, 1.1731847524642944, -2.1970489025115967, -0.5763148069381714, 0.3202402889728546, -0.5073438286781311, -0.16042768955230713, -1.1682196855545044, 0.8207167387008667, -0.5986215472221375, -0.13149242103099823, 0.47213226556777954, 0.9919019937515259, -0.279384046792984, -0.4410824775695801, -1.8438352346420288, 1.6604139804840088, -1.419061541557312, -1.6139923334121704, -0.6844088435173035, -0.06685622781515121, -0.1469423919916153, 0.926181972026825, -0.5178253650665283, 0.3247013986110687, -2.023345947265625, -0.23772463202476501, -0.3466958701610565, -0.908903956413269, 0.05385809764266014, -2.395277976989746, 1.100904941558838, -0.6868807077407837, -1.1252847909927368, -0.6450372338294983, -0.14783763885498047, -1.0652815103530884, 1.7087361812591553, 0.1301271617412567, -0.43169885873794556, 1.4643431901931763, 0.9794224500656128, -1.3627492189407349, -0.4457978904247284, 1.9622989892959595, -1.2563097476959229, -0.14719495177268982, -1.1722049713134766, -0.02056249789893627, -0.007088535465300083, 0.752297043800354, 0.5489342212677002, 1.8367220163345337, 0.488198459148407, -1.9645153284072876, -0.235646590590477, -1.4247137308120728, -0.4013373851776123, 1.8233656883239746, -3.382016181945801, -1.2399479150772095, -1.449475884437561, -1.159948706626892, 0.5323789715766907, -1.3902662992477417, 1.2114506959915161, -0.3370349705219269, -0.02613857574760914, 0.525543212890625, -1.3692151308059692, 0.29368317127227783, 0.07472732663154602, 1.0632715225219727, 0.4175914227962494, 0.15473313629627228, -0.9275524020195007, -0.13618528842926025, -0.015963150188326836, -0.47459203004837036, -0.1320507675409317, -0.5807825922966003, -0.7843719720840454, -1.1790790557861328, -0.494025319814682, -0.63944411277771, 2.4959957599639893, -1.0925573110580444, 0.019280510023236275, -0.8888875246047974, -0.15810434520244598, -0.405880331993103, 0.8711761832237244, 1.1413121223449707, -0.7441083192825317, 0.05446413531899452, 0.14615975320339203, -0.2272353619337082, 1.7090195417404175, 0.7934218049049377, -0.5766168236732483, 0.5416702032089233, 1.11550772190094, -1.5769665241241455, -0.6993358731269836, 0.6618317365646362, -0.6667434573173523, 0.0035804794169962406, -0.27355650067329407, -0.9486223459243774, -0.7926086783409119, 1.0069113969802856, -0.5582712292671204, -0.7165212035179138, 0.5409067869186401, 0.8644523620605469, 0.3552223742008209, -0.2831232249736786, -0.08703036606311798, 0.3205070197582245, 0.043878521770238876, 1.8781918287277222, -0.6810846924781799, 0.6754551529884338, 0.8744324445724487, 0.8582370281219482, -0.6064749956130981, -0.9241235256195068, -1.960342526435852, -0.12231212854385376, 0.6175857186317444, 0.3610444962978363, 0.5085133910179138, 0.9672448039054871, -2.225372791290283, -0.46094250679016113, 1.0661284923553467, -0.6644903421401978, 1.4153815507888794, -0.1549256145954132, 0.7403602004051208, 2.166086196899414, 0.7739607095718384, 1.8444353342056274, -2.2650153636932373, 1.1210927963256836, -0.5579332709312439, -1.2200490236282349, 1.681307315826416, 1.2048611640930176, -0.7008392810821533, 0.47906845808029175, -0.95070880651474, -0.17286106944084167, -0.4233289062976837, -0.2726019024848938, 0.38697734475135803, 0.03261025622487068, 0.5543392300605774, 0.6370851993560791, 1.218505620956421, -1.0419620275497437, -1.0729689598083496, 1.8489748239517212, 0.8686848878860474, 1.5436750650405884, -0.594126284122467, 1.1933821439743042, 1.368924856185913, -0.6965278387069702, -0.7873873710632324, -1.4842740297317505, -0.47821757197380066, -1.284840703010559, -1.28227961063385, 0.08393169939517975, -0.10634321719408035, -0.42588913440704346, -0.0374968983232975, -0.5165245532989502, -1.1516438722610474, -1.1109504699707031, 0.0866597518324852, 0.973187267780304, 0.8611288666725159, 0.1791669726371765, -0.12802492082118988, 0.8630781769752502, 2.3126862049102783, 0.5745925307273865, -1.3292906284332275, -0.10980416089296341, 0.5684522986412048, 1.584532380104065, -1.0230953693389893, 0.3038058578968048, -1.0300196409225464, -1.1490246057510376, 0.8162515759468079, 1.7736179828643799, 1.681398868560791, 1.1393433809280396, 0.48791876435279846, -0.45309731364250183, 0.5261190533638, -0.8652490973472595, 0.729677140712738, -0.8138886094093323, 1.3093236684799194, 1.2016243934631348, -0.7723386883735657, -1.3245775699615479, -0.46705663204193115, 0.07913974672555923, 0.766860842704773, -1.0816552639007568, 0.04910578206181526, 0.6344655752182007, -0.7164945006370544, 0.24233099818229675, -0.7027043104171753, -1.598358392715454, -0.7501407861709595, -0.4022538661956787, -1.6984657049179077, 0.3513419032096863, -1.8241568803787231, 1.4586186408996582, 0.634459912776947, 1.4309078454971313, -0.25947457551956177, 0.4499295651912689, 1.81901216506958, -0.05306585878133774, 0.4901689291000366, -0.6278725862503052, 2.096252679824829, -0.3713214695453644, -1.6432240009307861, -1.6107081174850464, -1.6375298500061035, -0.7586376070976257, 0.0758991613984108, 1.2183544635772705, -0.9872817993164062, -1.9598290920257568, -0.7314902544021606, -2.4535624980926514, -0.07147449254989624, 0.6604222655296326, 0.1710342913866043, 0.7541255950927734, 2.458707094192505, -1.2153029441833496, 1.9794321060180664], [-1.4017239809036255, -0.39690229296684265, -2.190288782119751, -0.25815245509147644, -0.05907606706023216, -0.2311834841966629, -0.1932581663131714, -0.1290583610534668, -1.0418983697891235, 0.7978467345237732, 0.12658186256885529, 1.0746958255767822, -0.37389522790908813, 0.27300211787223816, -1.4080060720443726, -0.9670973420143127, -0.9261570572853088, 0.9430019855499268, 0.6626257300376892, -0.24302376806735992, 0.10586214810609818, -0.35497620701789856, 2.0626273155212402, -0.9956274628639221, -1.392220377922058, -0.5245497822761536, 0.7943746447563171, -1.5807459354400635, 1.0468688011169434, -0.705682635307312, 0.05146538466215134, -0.20325778424739838, 1.1228556632995605, 1.423664927482605, 1.645855188369751, -0.8382120132446289, 0.7944139242172241, -0.99757319688797, -0.7358852028846741, 1.0639606714248657, -0.7672406435012817, -0.325224906206131, 0.03512139245867729, 0.2906728684902191, -0.6517512202262878, 1.3706210851669312, -1.2931878566741943, -0.23600468039512634, 0.6576246619224548, -0.6773623824119568, 0.8036735653877258, 0.9299245476722717, 0.22027909755706787, 0.1548614203929901, -1.4304463863372803, -0.9982934594154358, 0.19309864938259125, -1.0934900045394897, 1.0840057134628296, 0.7382954359054565, 1.8940082788467407, 0.616922914981842, -0.3239089548587799, 1.7080087661743164, -1.4083391427993774, 0.2023804783821106, -0.6190727353096008, 1.4407044649124146, 1.0953303575515747, -0.053550634533166885, -0.3019851744174957, 1.259925365447998, 1.1698991060256958, 0.09474552422761917, 1.0698531866073608, -0.016563400626182556, 2.35510516166687, 1.6351814270019531, -1.2783312797546387, -1.0429868698120117, 0.14974583685398102, 1.0506912469863892, 1.1572352647781372, 0.022060707211494446, 0.752001166343689, 1.6529344320297241, 1.1816496849060059, -0.36211028695106506, -0.5669309496879578, 0.21207883954048157, -0.6479759812355042, -0.7111843824386597, -0.22469596564769745, -1.1277563571929932, -0.5559848546981812, 1.1703451871871948, -1.9337854385375977, -0.6469800472259521, 1.3899483680725098, -0.6532682180404663, -1.0970799922943115, -0.6266646981239319, -0.10853225737810135, -1.4710997343063354, -0.9332003593444824, -1.1145821809768677, -0.368455171585083, -0.4669495224952698, -1.649635910987854, -0.19562393426895142, -0.6743202209472656, -0.2488640695810318, -1.1293307542800903, 2.231536388397217, 1.4777170419692993, 0.4753487706184387, 0.4847346246242523, -0.08833098411560059, 1.3339674472808838, -0.7561197280883789, -0.3469526767730713, -1.0676511526107788, 0.031427014619112015, -0.4224811792373657, 0.4985668957233429, -0.300354927778244, -0.2363707423210144, -0.8835757374763489, -0.12631836533546448, 0.16209188103675842, 1.1161900758743286, 1.688890814781189, -0.40788111090660095, -2.1091480255126953, -0.9114063382148743, -0.7941678166389465, 1.961172342300415, 0.08210915327072144, -0.8589422702789307, -1.2947821617126465, -1.0940581560134888, 0.6602985858917236, 0.8885259032249451, 0.1913897842168808, -0.8704214096069336, 2.6292238235473633, 1.6755067110061646, 0.09561356902122498, -0.5470620393753052, -1.4616566896438599, 0.3936085104942322, 0.7331811785697937, 1.281306266784668, 0.6692151427268982, 1.8332407474517822, 0.375785768032074, 0.9181393980979919, 0.5169808864593506, 0.2339307814836502, 1.88205885887146, 1.4002397060394287, -0.926738440990448, 1.0300074815750122, 0.033759795129299164, 1.0769851207733154, -0.4280247986316681, 2.567250967025757, -1.1835582256317139, 0.3629853427410126, -0.25199195742607117, 0.6019536256790161, -1.5743328332901, 0.11501257121562958, 0.3524293899536133, -0.599555253982544, 1.8360931873321533, 0.8901252150535583, 0.4897077977657318, -0.649884045124054, 0.8919274210929871, -0.07189887762069702, 1.7441500425338745, 0.1575648933649063, 0.08126532286405563, -0.25332820415496826, 0.908103346824646, 1.4016200304031372, -0.050622571259737015, 0.3183246850967407, -0.1634320318698883, -0.205221027135849, 1.6005949974060059, 0.046164512634277344, -0.8421409130096436, 0.14196988940238953, -2.096646547317505, -1.183616042137146, -1.938599705696106, -0.868701696395874, 1.9688329696655273, 0.17859132587909698, 0.7414045929908752, -1.5716705322265625, 1.9309139251708984, 0.3871903121471405, -2.599886655807495, -2.240154981613159, 0.5814136266708374, -2.485322952270508, 0.1257573366165161, -0.9264615774154663, 0.399966835975647, -0.12506146728992462, -1.031283974647522, -0.11912491172552109, 0.785797655582428, -1.0561046600341797, 0.5948116183280945, -0.6638191938400269, 0.8833242654800415, -0.274651437997818, 1.6166218519210815, -0.02425648830831051, -1.410151720046997, 0.9763143062591553, 1.5544947385787964, 1.8300694227218628, 0.3320017158985138, -2.053158760070801, 1.119099497795105, -0.20265892148017883, -0.33977770805358887, 1.4746935367584229, 0.06387665122747421, 0.061179373413324356, -1.2245063781738281, -0.9196543097496033, -1.00113844871521, -0.6324350237846375, -0.558211624622345, 0.7728943228721619, -1.4677320718765259, -0.6450733542442322, 1.863868236541748, -0.03483650088310242, 0.8366196751594543, -0.045245058834552765, -1.7411108016967773, 0.7730551362037659, 0.21071265637874603, -0.4155953526496887, -0.12023365497589111, -1.0799576044082642, -0.003645131830126047, 0.7523832321166992, -0.38333532214164734, -0.36634230613708496, 1.8416697978973389, 0.3586958050727844, 1.9400473833084106, -1.635699987411499, -1.8500546216964722, 0.20118969678878784, 0.5536969304084778, 0.24141648411750793, -2.3359482288360596, -0.8929417729377747, 2.0251195430755615, 0.3812704384326935, 1.879132866859436, -0.2123783528804779, -1.087110996246338, 0.5512032508850098, 0.26329782605171204, -0.1420314460992813, 0.8613144755363464, -0.5638993382453918, 0.04100465774536133, 0.8867184519767761, -0.15807634592056274, 1.1624654531478882, -1.0805004835128784, 0.6238000392913818, -1.3005796670913696, 1.8375567197799683, -1.482500433921814, -0.06795518100261688, -0.00913893524557352, 0.9068246483802795, 0.257199764251709, -1.4570759534835815, -0.42583751678466797, -2.0001542568206787, -1.2988662719726562, -1.3497231006622314, -0.3283516466617584, -1.1922246217727661, -2.460538387298584, -0.8496191501617432, 1.5280638933181763, 1.0753639936447144, 0.5009452700614929, -0.027099953964352608, -0.6127195358276367, -0.39171966910362244, 0.5195019245147705, 0.5988913774490356, 0.9034652709960938, -0.6765578389167786, 1.0117684602737427, -0.7013312578201294, 1.659512996673584, 1.2298095226287842, -0.7597220540046692, 2.265451431274414, 0.25035393238067627, -0.5419577360153198, -1.0345641374588013, -0.6342340707778931, -0.5670722723007202, 0.372275710105896, 0.26306894421577454, 0.3383057415485382, -1.3136712312698364, -0.4670715630054474, -0.6724891066551208, 0.038376279175281525, 0.604224681854248, 0.6031133532524109, -1.9917842149734497, 0.1327424943447113, -1.057247281074524, -0.9054953455924988, -0.42717230319976807, 0.23770609498023987, 0.3858107924461365, -0.45541033148765564, -0.33463993668556213, -0.1906081587076187, 0.5983690023422241, -0.062210455536842346, 0.31981411576271057, 0.725398600101471, 0.35126471519470215, 0.1012895479798317, -0.2816980481147766, 0.3771703541278839, -1.1103782653808594, -0.5560681819915771, -0.2973167300224304, -1.6806340217590332, 0.9783070087432861, -1.307509422302246, -1.1829493045806885, 1.0203444957733154, -0.6197026968002319, -1.088365077972412, -1.0319619178771973, 0.9910743236541748, 2.151667356491089, 0.3173690736293793, -0.21809421479701996, -0.28527095913887024, 1.2509148120880127, 0.5301478505134583, 0.03865892067551613, 1.3531726598739624, 0.17527686059474945, 1.5345799922943115, -0.5341176390647888, -1.131661295890808, -0.7636744976043701, 0.5507205128669739, -1.6072343587875366, 0.3031718134880066, 0.6242004036903381, 0.30902665853500366, 1.1716793775558472, -0.9167978763580322, -0.4678899943828583, 0.637050986289978, -1.3414775133132935, 0.541424036026001, -0.6336165070533752, 0.9926760196685791, 0.4130043685436249, 0.9860410094261169, -0.5693602561950684, 0.07914909720420837, 2.385885238647461, -0.457283079624176, 1.3717150688171387, -0.9344755411148071, -0.3882787525653839, 1.1604905128479004, -1.2155134677886963, 0.8695643544197083, -0.007135747000575066, -0.008898247964680195, -0.23519179224967957, -1.1980713605880737, 1.503077507019043, 0.8915289044380188, -1.6871943473815918, -0.2538231313228607, -0.036056745797395706, 0.45257383584976196, -3.012756586074829, -0.3318588137626648, -0.14005284011363983, 0.3682076632976532, 1.207998275756836, 0.3012309968471527, -2.5152573585510254, 2.2346513271331787, 0.8849549293518066, -1.007025122642517, 0.41440752148628235, -0.8963744640350342, -0.0722445547580719, -0.4828417897224426, -0.6034550666809082, -0.4313644468784332, -0.5497514009475708, -1.3832310438156128, 1.7730525732040405, 0.38863202929496765, 0.797878086566925, -1.0126550197601318, -0.041456907987594604, 2.241623640060425, 0.12439525872468948, 0.5566107630729675, 1.134477138519287, -0.20845063030719757, 0.8566754460334778, -0.9953914284706116, 1.1630268096923828, -1.256049633026123, 0.6554192900657654, 0.44921454787254333, -0.6148030161857605, -0.39201027154922485, 1.1157885789871216, 0.06501897424459457, 0.19712477922439575, -1.9528831243515015, 0.36893537640571594, -0.12131483107805252, 0.15961627662181854, -0.6271891593933105, -1.70357084274292, -1.1157606840133667, -0.6121329665184021, -0.3931739628314972, 0.9693095088005066, 1.123060941696167, 0.11307666450738907, 0.7472368478775024, 1.0113469362258911, -1.2010674476623535, -0.9322654008865356, 0.9198543429374695, 0.36244910955429077, -0.49646642804145813, -0.41921544075012207, 0.6610888838768005, -0.7673571705818176, -0.0029771472327411175, -0.920010507106781, -0.07118858397006989, 1.356863260269165, -0.357710063457489, -2.6572015285491943, 1.1226886510849, 0.1901172548532486, -1.1845405101776123, -0.16979250311851501, 1.0186457633972168, -0.514682948589325, 0.9194093942642212, -1.4195352792739868, 1.4726886749267578, -1.6702966690063477, -0.5830802917480469, 1.0893661975860596, 0.9450442790985107, 1.2211395502090454, -0.5286104679107666, -0.3164670467376709, 0.7382677793502808, 1.482080340385437, 0.9630604386329651, 1.1336528062820435, -1.4257879257202148, -0.7238615155220032, -0.12996169924736023, 0.30021244287490845, 0.8439998626708984, 0.3928583860397339, -0.43367084860801697, 0.7019885778427124, -0.4147515594959259, 0.14227180182933807, 0.422194242477417, -0.8380282521247864, 0.5291628241539001, -1.2188048362731934, -0.9333962202072144, 0.4270349144935608, -0.05043138191103935, 0.478148877620697, 1.2417423725128174, 0.017123330384492874, -1.0358028411865234, -0.5408667325973511, 1.0617737770080566, 0.9606322646141052, 0.38047051429748535, 0.05350729078054428, -0.4131423234939575, -0.6791713237762451, 1.0242953300476074, -1.3225419521331787, -0.8397429585456848, -0.6191229820251465, -0.055422019213438034, -1.1178697347640991, 1.9837064743041992, 1.6534205675125122, -0.7489569187164307, 0.1392275094985962, -0.6936818957328796, 0.030568651854991913, 0.12484758347272873, 0.5745647549629211, 1.0405805110931396, -1.0927975177764893, -0.4142930805683136, 1.4666638374328613, -0.9077726602554321, -1.2935212850570679, 0.34350669384002686, -0.58806312084198, -0.8718401789665222, -0.5533840656280518, -0.7603204846382141, 0.22067150473594666, -0.08284814655780792, 1.3729960918426514, -1.1276209354400635, -0.3010101020336151, -0.49685171246528625, 0.6281716823577881, 1.4336079359054565, 0.7869083285331726, -0.15127970278263092, 0.6878740191459656, -0.3723263442516327, 0.40952998399734497, -1.4332515001296997, 0.04215589165687561, -0.0824369490146637, -0.13751035928726196, -1.2184029817581177, 0.5660645365715027, 0.2387804090976715, 1.3570789098739624, 0.49754565954208374, 1.651658535003662, -0.22679319977760315, -0.4315527379512787, 1.0665045976638794, 0.4668799638748169, -0.8201695084571838, 1.0016674995422363, 0.5969970226287842, -0.6483302712440491, 1.8127942085266113, -0.6102392077445984, 0.22732508182525635, 0.15789884328842163, 1.8459104299545288, 0.3762844204902649, -0.3002013862133026, -1.575957179069519, 0.33423447608947754, 0.2528277635574341, -1.2996352910995483, -0.6821531057357788, 1.2363336086273193, -0.14733290672302246, 0.8311758041381836, 1.691691279411316, 0.6513829827308655, 0.8611754775047302, 0.7137773036956787, -0.9494109153747559, 0.18135175108909607, 0.34339895844459534, -0.3474549651145935, 1.1235519647598267, 0.5995149612426758, 0.8090317845344543, -0.06566490978002548, -0.650354266166687, -0.5903939604759216, 0.12194553762674332, -1.0134984254837036, 2.0170233249664307, -0.16296061873435974, 0.1864626258611679, 0.8810042142868042, -1.42145574092865, -0.4388566315174103, -0.47364354133605957, 1.0855975151062012, 0.10930033773183823, 0.2688222825527191, 0.17164841294288635, 0.6006789803504944, -1.7137832641601562, -0.6665256023406982, -0.43208521604537964, -0.8593096733093262, 1.3274210691452026, -0.8712154626846313, 0.2941378951072693, 0.5152938365936279, 0.5117392539978027, 0.11282767355442047, -2.493062734603882, 1.033128023147583, 0.34998828172683716, 1.0495964288711548, -1.1866872310638428, 0.6343660354614258, -1.0148597955703735, 0.5602541565895081, -1.6546530723571777, -0.19261117279529572, -1.354280948638916, -0.7850131392478943, 1.4823024272918701, -0.9923874139785767, -0.05168457701802254, -0.6878644824028015, -0.3595774471759796, 1.6359999179840088, -1.2480701208114624, -0.3402700424194336, -1.1095538139343262, 1.327660083770752, 0.3496609330177307, -1.127716064453125, 0.23793575167655945, 2.629007577896118, 0.28476008772850037, -1.4225877523422241, -0.5324673056602478, 0.8299337029457092, -0.4786261022090912, -0.2488659918308258, -0.05128977447748184, -1.2878283262252808, -0.45107340812683105, -1.1559098958969116, 0.8749176859855652, 1.2293413877487183, -0.5237801671028137, -1.3936480283737183, 0.6348690986633301, 1.0092734098434448, 0.3201414942741394, 0.3072134256362915, 1.611345887184143, -0.589068591594696, -0.21811877191066742, -0.639528214931488, -0.07740303874015808, 0.2634996473789215, -0.2410692721605301, -0.5439742207527161, 0.8484364151954651, 1.0400642156600952, -0.09225241839885712, 1.6236107349395752, -0.3750753700733185, 0.130431666970253, 2.4941868782043457, -0.025451278313994408, 1.5282948017120361, 2.431575059890747, -0.25300103425979614, 0.9265164136886597, -0.7185924649238586, 1.5995925664901733, 0.4018568992614746, -0.027932271361351013, -0.7349417209625244, 0.8937773108482361, 1.3437186479568481, -1.589698314666748, 0.40943291783332825, 0.34730246663093567, 0.8157006502151489, -1.2880825996398926, 0.22001726925373077, -0.23580586910247803, 0.24485543370246887, -0.5965795516967773, 0.8596295118331909, -1.1603081226348877, -1.1806703805923462, 1.2850185632705688, -0.49346768856048584, -0.6518743634223938, 0.3153076469898224, 2.1192095279693604, 0.5882251858711243, 1.0596269369125366, -1.4963927268981934, -0.39085882902145386, 0.0061503127217292786, 0.14740076661109924, -2.3098080158233643, 1.168368935585022, 0.880259096622467, 1.9360404014587402, 0.10759555548429489, -1.1225311756134033, 1.1157691478729248, -1.1593893766403198, 0.08287830650806427, -0.9168362617492676, -0.8041950464248657, 0.3278367221355438, -0.279047429561615, -1.2957758903503418, -0.13937142491340637, -0.5616634488105774, -1.5045245885849, -1.2223833799362183, -0.30063822865486145, 0.09751330316066742, -0.24126316606998444, 0.7777209281921387, 0.7459421753883362, 0.5984804630279541, 0.15358708798885345, 1.3748892545700073, 1.2271134853363037, 1.270447850227356, 0.44349607825279236, 0.4710943102836609, -0.9602217674255371, -0.5360840559005737, 2.136220693588257, -0.08248436450958252, -0.19024567306041718, 0.6707772016525269, 0.28310391306877136, -1.5448256731033325, 0.8920562267303467, 0.3179793357849121, 0.8283109068870544, -0.14396759867668152, -0.8963225483894348, 0.6272872090339661, -0.29402369260787964, -0.16336503624916077, -0.6465712785720825, 0.9864774346351624, 0.6786867380142212, -0.5033509731292725, -0.4855493903160095, 0.7557814121246338, 0.8901591300964355, 0.8107510209083557, -0.583143949508667, -0.4524424076080322, -0.9031206369400024, 0.22145842015743256, 1.1609987020492554, 1.0674562454223633, -0.9477256536483765, -0.930880069732666, 2.2008297443389893, -0.8082956671714783, 0.16014479100704193, -0.4731091558933258, 1.516588807106018, -0.7130748629570007, 0.8042666912078857, 0.5629559755325317, 0.7023048996925354, -0.4674825072288513, -1.7189449071884155, 0.28808486461639404, 0.15852877497673035, -0.9692413806915283, -0.36141830682754517, 1.1432349681854248, -1.3781459331512451, -0.2705112397670746, 0.5744125843048096, -1.986717939376831, 0.8997487425804138, 0.22469736635684967, -0.4795420467853546, 0.18487799167633057, 1.4340863227844238, 1.0277620553970337, -0.31319913268089294, -1.9101327657699585, -0.32263705134391785, -0.5871832966804504, -0.8055235147476196, 1.1465444564819336, -1.9661555290222168, 0.4194434583187103, -1.3281913995742798, 0.9876271486282349, -0.2416035532951355, 0.7137491703033447, 0.05956336110830307, 0.003483841195702553, -0.5704295039176941, 1.0644174814224243, -0.8678352236747742, 1.353991985321045, 0.15133975446224213, 0.30078887939453125, -1.3108303546905518, -0.9232642650604248, -0.034920867532491684, 1.0434753894805908, 1.3927879333496094, 0.21525916457176208, 0.5576146841049194, 1.951493740081787, 1.7190301418304443, -1.1255879402160645, 0.8431720733642578, -0.9883447289466858, -2.18815016746521, -2.226696014404297, -0.02560495212674141, -1.0547475814819336, 0.48759374022483826, -0.12504424154758453, 0.7049807906150818, -0.4883033335208893, -1.054824948310852, 0.10585435479879379, 1.0232093334197998, 0.07777758687734604, -0.8488919138908386, -0.5448299050331116, 1.0287611484527588, -1.208715796470642, 1.4995149374008179, -0.13381758332252502, -0.17796333134174347, 1.5307916402816772, -0.352322518825531, 1.3611582517623901, 0.20435155928134918, 0.2612735331058502, 0.1581430733203888, 0.5093845725059509, 1.5707032680511475, 0.8137957453727722, -1.188265323638916, -1.34067964553833, -0.7719027400016785, 1.816415786743164, -0.38547074794769287, 0.5830371379852295, -1.2679797410964966, -0.4607045650482178, -0.3939022719860077, 1.1820417642593384, 1.5629656314849854, 1.8494970798492432, 1.1620864868164062, 0.47105470299720764, 0.7503960728645325, 0.4688560664653778, -1.3192390203475952, -2.103883743286133, -0.1344333440065384, -0.28673481941223145, 1.8477195501327515, 1.3855268955230713, -2.308724880218506, -1.1056334972381592, -2.0886592864990234, -0.29711756110191345, 1.0403761863708496, -1.4491312503814697, 0.8801254034042358, -1.1525176763534546, -1.1036622524261475, -0.6601520776748657, -2.929809331893921, -0.3277112543582916, 0.5464692711830139, 1.3558149337768555, -0.2565363347530365, 0.3979693353176117, 0.8678461909294128, 1.128502607345581, -0.06134895980358124, 2.2011260986328125, -1.5388853549957275, -0.09844491630792618, -0.22388099133968353, -0.6121544241905212, 1.130094289779663, -0.2244148701429367, -0.9237096905708313, -0.6612334251403809, 1.0130034685134888, 0.6424210667610168, -0.37778738141059875, -0.2516235411167145, 0.8348570466041565, -0.6030938029289246, 0.5401809811592102, 1.3957864046096802, -1.9759758710861206, -1.1084843873977661, 0.8483966588973999, -0.2278113216161728, 0.3806072473526001, 0.18004076182842255, -0.4816216230392456, 0.6303744316101074, -0.3828183710575104, -1.1224952936172485, -1.099951148033142, 0.15414835512638092, 0.3349960744380951, 0.31984180212020874, -0.9076077342033386, 0.8190905451774597, -0.5706100463867188, 0.8276326060295105, 0.7720265984535217, 0.9929504990577698, 1.5275174379348755, -1.5427592992782593, -0.9884625673294067, 0.4578753411769867, 1.6997957229614258, 0.8814504146575928, -0.13522736728191376, 0.045254185795784, 0.08873052150011063, -0.2937432527542114, 0.2035689651966095, 0.24061882495880127, -0.48993003368377686, 1.7188329696655273, 0.6485887169837952, 0.7888332009315491, -0.3467286229133606, 0.2734330892562866, -1.3610221147537231, -0.1374579519033432, -0.784805417060852, -0.2552714943885803, 0.4661387503147125, 0.987980306148529, 0.5788918137550354, -1.128818154335022, -0.10019354522228241, -0.39254260063171387, -0.09531945735216141, -0.0654015988111496, -2.5921688079833984, -0.25856536626815796, -0.5127455592155457, 0.22957967221736908, 0.47387272119522095, 1.2135610580444336, -1.4792590141296387, 1.4363569021224976, 0.7506622076034546, 1.3928627967834473, 0.06049215421080589, 0.7451928853988647, 0.5871548056602478, -0.1112971231341362, -0.7290078997612, 0.7987961769104004, 0.8431118726730347, 1.6546708345413208, 0.2316778004169464], [1.359245777130127, 0.12982046604156494, -1.3349028825759888, 0.6092426180839539, -1.4301321506500244, -0.17520996928215027, 1.0657382011413574, -0.17739734053611755, -0.2879118025302887, -1.5954852104187012, 1.0838415622711182, 0.5326504707336426, -1.9213905334472656, -0.3346301317214966, -0.07721967995166779, 0.29416805505752563, 1.2212902307510376, 0.9506024718284607, 0.09954485297203064, 2.1140339374542236, -0.08477070927619934, 0.20139449834823608, 0.20265427231788635, 0.9842507839202881, 1.1485038995742798, 0.40942153334617615, 2.160372495651245, -0.04500633850693703, -1.1113717555999756, -0.6900268793106079, 0.939242422580719, 2.3562750816345215, 1.2466894388198853, -0.28994619846343994, -0.6624544262886047, 1.3787800073623657, 1.1901227235794067, 0.19061364233493805, 1.3251826763153076, -1.4188591241836548, 0.3812274634838104, 0.2101995199918747, 0.8293268084526062, -0.3692548871040344, 0.3789574205875397, 0.08481016755104065, -0.6589205265045166, 0.17907124757766724, 1.7360588312149048, 0.969076931476593, -0.6950851678848267, -0.5259456634521484, -1.4781831502914429, 0.14814014732837677, -2.154578924179077, -3.9982238376978785e-05, 0.4215197265148163, -0.7031887173652649, 0.8261222839355469, 0.10409235209226608, 1.2894055843353271, -1.0999913215637207, -2.0180702209472656, 0.12183742225170135, 0.22437025606632233, 0.8113405108451843, -0.3568171262741089, -1.1801376342773438, 0.05618605762720108, 1.4762500524520874, 2.0249791145324707, 1.455264687538147, 0.1695708930492401, -0.4528689682483673, 0.0383571982383728, -0.8755290508270264, -1.2794909477233887, -1.463957667350769, -0.3118517994880676, -1.0156316757202148, 0.24867622554302216, -0.8058414459228516, -1.1197224855422974, -1.9192450046539307, -0.9308123588562012, 0.5177412033081055, 1.3089348077774048, -0.19572660326957703, -0.3908902108669281, -1.764158010482788, 1.1009281873703003, 0.27949318289756775, -1.09702730178833, -0.016947194933891296, -0.2992247939109802, 0.5151562690734863, 0.41265133023262024, 0.6532880663871765, 1.3258256912231445, -0.6367472410202026, 1.3426681756973267, 0.552992582321167, -2.4123003482818604, -2.5988316535949707, -0.39110711216926575, 0.23034557700157166, -0.6095800995826721, -1.6552903652191162, 0.16699913144111633, 1.7751635313034058, -0.041581347584724426, 0.2827047109603882, -0.994118332862854, -0.05082155019044876, 0.9269047379493713, -0.2301573008298874, -0.3654709458351135, 1.2134703397750854, 0.8067225813865662, -0.31667396426200867, -1.1574877500534058, -0.11802281439304352, -1.833918571472168, -0.6087818741798401, 1.1096152067184448, 0.4700826406478882, -0.9077860713005066, -1.241500973701477, 0.4152568578720093, 1.194865345954895, -1.3261767625808716, 0.2640922963619232, 0.8637785911560059, 2.661975383758545, 0.5888833403587341, 0.38985732197761536, -1.152085542678833, 0.9164484739303589, -0.03034345805644989, -0.12769463658332825, -0.26232215762138367, -0.2023046612739563, -1.0743683576583862, 1.000378966331482, 0.4303191304206848, 0.8191303610801697, -1.0032167434692383, -0.9654257893562317, 0.4232548475265503, 1.2127240896224976, -0.3561660647392273, -0.8530420064926147, 1.2820234298706055, -1.510948657989502, -0.22532597184181213, 2.7302136421203613, -0.39012160897254944, -0.4014109671115875, 0.21135765314102173, 0.33392760157585144, 0.6346338391304016, -0.03912067785859108, -0.9649900794029236, 0.2861056327819824, 0.7537248730659485, -0.909432590007782, -0.6601461172103882, -0.7373361587524414, 0.05642211437225342, 0.684661328792572, -0.7381644248962402, -0.1775205135345459, 0.6122221350669861, -0.5552263259887695, -0.5764349102973938, -0.24435117840766907, 0.43685245513916016, -0.6270401477813721, -0.5969552993774414, -0.07348249852657318, 0.23209501802921295, 0.8033743500709534, 2.3268606662750244, 0.04839673638343811, 0.47964349389076233, -0.010687526315450668, 1.0851614475250244, -0.16075414419174194, 0.25792786478996277, 0.02266443334519863, -0.4651736617088318, 0.06104804947972298, 0.16708172857761383, -1.6892045736312866, -0.7108321785926819, -0.5843530893325806, 0.9367321133613586, 0.5579400658607483, 1.523151159286499, 0.7622029781341553, 1.013185977935791, -1.2851618528366089, 0.07478569447994232, -1.04346764087677, -1.3039882183074951, -1.047316074371338, -1.1420013904571533, -0.5923140645027161, 0.7484226226806641, 0.7046452164649963, -3.3334593772888184, 0.9825025796890259, 0.6974020600318909, 0.813154399394989, -0.5239418745040894, -1.107452630996704, 1.2780373096466064, -0.24601732194423676, 0.5807015895843506, -0.3884967863559723, -0.4478696882724762, -0.21628284454345703, 0.38150858879089355, 1.5341418981552124, -1.0450843572616577, -0.6405475735664368, -0.4794524610042572, -0.9729302525520325, -0.14943605661392212, 1.282366156578064, 0.25136512517929077, -1.3190127611160278, 2.5683393478393555, 0.17081446945667267, -1.1950976848602295, 1.6567859649658203, -0.6060745716094971, 0.5178601741790771, 1.917377233505249, 1.1520886421203613, 0.8880141973495483, 0.12086760997772217, -0.042140502482652664, 0.3254331648349762, 1.0412753820419312, -0.6158915758132935, -1.500802755355835, -0.425392210483551, 0.48351454734802246, 0.5506682395935059, 1.5006098747253418, -1.441687822341919, -0.8867942690849304, -0.5439961552619934, 0.4137711226940155, -0.1463390588760376, -1.1395996809005737, -0.8290776014328003, 0.12638264894485474, 1.9829920530319214, 0.5436459183692932, 0.854788601398468, -0.08074139058589935, 1.1179111003875732, 0.07239208370447159, -1.3606905937194824, -0.5466980934143066, -0.8266592025756836, 0.8321982026100159, 0.0796269029378891, -0.22433282434940338, 1.3786799907684326, -1.7711337804794312, 1.5907831192016602, 0.6144980788230896, -1.1362472772598267, -1.6605806350708008, 0.3180968463420868, 0.09855969995260239, 0.7306694388389587, -0.8513286113739014, -0.49014797806739807, -0.5349704623222351, -0.016405262053012848, 1.8060520887374878, -0.8570418357849121, 0.056789834052324295, -0.879352331161499, 0.21272875368595123, 1.0049854516983032, -0.36380332708358765, 0.3602820932865143, 0.8736848831176758, 0.5884914398193359, -0.7804804444313049, -0.4646591246128082, -0.6526919007301331, 0.6598503589630127, -2.1371471881866455, 1.1135042905807495, -0.24842365086078644, 2.5446126461029053, -0.9594572186470032, 1.3330409526824951, -0.9464399218559265, 1.5363800525665283, 0.2620367705821991, 1.154252052307129, 1.128495454788208, 0.5706565380096436, -0.7801297903060913, -0.92637699842453, -0.7498297095298767, 0.06543096154928207, -0.25564366579055786, -1.2200517654418945, -0.45122864842414856, 3.171022415161133, -0.889926016330719, -1.0634512901306152, -0.6241495609283447, -1.426766276359558, -1.504544734954834, 0.10816217958927155, -0.4248770475387573, -0.9593272805213928, -0.23320624232292175, -0.7997189164161682, 0.5569157600402832, -0.13734497129917145, -0.14139041304588318, 0.6288456320762634, -1.0830694437026978, -0.5973303318023682, 2.452745199203491, -1.3400652408599854, 1.2277275323867798, 0.12327531725168228, 0.6431072354316711, 0.3712003231048584, 0.1072373017668724, -1.1278709173202515, 0.9201586246490479, -0.30745112895965576, -0.3950086534023285, -1.3931869268417358, 1.0552188158035278, -0.3843194246292114, -1.2799270153045654, -0.21279892325401306, 0.6701550483703613, 0.5800440907478333, -2.1125359535217285, 1.5212652683258057, 1.0012257099151611, 1.0412020683288574, -0.011149981990456581, -0.1945481300354004, -1.3217072486877441, -2.1765410900115967, -1.9068593978881836, 0.1896580159664154, 0.24388670921325684, 0.8542744517326355, -0.5911004543304443, -0.279016375541687, 1.1797090768814087, -2.533264636993408, -0.7535187005996704, -0.9681508541107178, -1.6641621589660645, -0.9963297247886658, -0.4483815133571625, -0.49694597721099854, -1.8630861043930054, -1.4308419227600098, -0.028155509382486343, -1.1241178512573242, 0.10628574341535568, -2.2301485538482666, 1.4146453142166138, 0.5033729672431946, 0.4659711718559265, -1.0942981243133545, -0.29200953245162964, 1.4717463254928589, -0.416786789894104, -0.38216516375541687, -0.625907301902771, 0.5752066373825073, 1.3078776597976685, 0.2604271173477173, -0.7329400181770325, 0.28869637846946716, 1.7989134788513184, -1.322568416595459, 1.5268384218215942, 0.10554095357656479, 0.9255445003509521, 0.7229470014572144, 1.1099317073822021, -0.0339837484061718, 1.6087085008621216, 1.370634913444519, 0.8054719567298889, -1.5269216299057007, 0.4375436007976532, -0.8635625839233398, -0.19585078954696655, 0.31209591031074524, -1.883863925933838, 1.639319658279419, 0.867883026599884, 0.7213341593742371, -0.27593356370925903, 0.1894826740026474, 1.8626667261123657, 0.20340904593467712, -1.7667617797851562, -1.6332883834838867, 0.1419382244348526, -1.3509254455566406, -0.39932748675346375, 0.3258580267429352, -0.06627258658409119, 0.2440890669822693, 0.4832600951194763, 0.26564422249794006, -1.102189540863037, 1.3717851638793945, 0.2875627279281616, -0.0012802394339814782, 1.2173864841461182, -0.13589735329151154, -0.6983069777488708, 0.3419676721096039, -1.224408745765686, 1.6292426586151123, 1.1863497495651245, -0.07275933027267456, -0.1213853508234024, -1.7778586149215698, 0.6963058114051819, 1.5678470134735107, -0.06312919408082962, 2.729072093963623, 2.9633939266204834, -0.15314249694347382, 1.2620766162872314, 1.4751142263412476, -0.3338654935359955, 0.8812016844749451, 0.3958837687969208, -0.300204336643219, 0.3432546854019165, 0.43316349387168884, -0.2993091344833374, 0.8080083727836609, -1.0276893377304077, -0.0865890309214592, -0.2894388437271118, 0.1092282384634018, -0.6398850083351135, 1.2367621660232544, 0.371599406003952, -0.542057991027832, -3.284454822540283, 0.05353491008281708, -0.7006506323814392, -0.9546424746513367, -0.14591947197914124, 0.291502445936203, 0.16004882752895355, 1.6182068586349487, 0.9921033978462219, 1.128420114517212, 0.49704471230506897, -0.4218906760215759, 0.1531505584716797, 1.05873703956604, 0.2629048824310303, 1.4216445684432983, 0.6505737900733948, -0.84328293800354, 1.6701728105545044, -0.9407266974449158, 0.43707504868507385, 0.19559156894683838, 1.5102438926696777, -0.7384624481201172, -0.03633785620331764, 2.0248782634735107, 1.1996136903762817, 0.8159898519515991, -0.9422184824943542, 0.3597460389137268, -0.40174299478530884, 0.543291449546814, 1.4579936265945435, 0.9138092398643494, -0.29794999957084656, 0.15045510232448578, 2.838768720626831, -1.8343861103057861, 0.9992955923080444, 0.6071851253509521, -0.9015598893165588, 1.8750483989715576, -0.45251962542533875, -0.11046701669692993, 1.6133334636688232, 1.3925589323043823, 0.2713887691497803, 0.5370676517486572, -2.043325901031494, 0.650087296962738, -1.5854270458221436, 0.45620158314704895, 0.42860502004623413, 0.23145632445812225, 0.4807797372341156, 1.1601989269256592, 0.39354056119918823, -1.1752220392227173, -1.6582262516021729, -0.22083687782287598, 1.0737804174423218, 1.2284165620803833, 0.1818322092294693, 0.27587535977363586, 1.8044090270996094, -0.4184759259223938, 1.4866799116134644, 0.5335529446601868, -1.3162801265716553, -0.8418068885803223, 0.8664396405220032, 0.7893922328948975, -1.7258105278015137, 0.4955301582813263, 1.5138211250305176, 1.4378135204315186, 1.0682123899459839, -0.435087114572525, -0.8234656453132629, 1.3498055934906006, 2.301344633102417, -0.1066538468003273, -1.7302186489105225, -0.7060843706130981, -0.12401247769594193, 0.06223384290933609, 0.9040377140045166, 1.368506908416748, 0.07547055929899216, 0.8084791898727417, 0.8759086728096008, -0.17380738258361816, 1.5863878726959229, 0.5905448794364929, 1.0324972867965698, -0.6418552994728088, 0.8403551578521729, 0.19780926406383514, -0.969352126121521, 0.5601333379745483, 0.8620921969413757, 1.845396876335144, 0.02096381224691868, 0.06613653898239136, -0.868121325969696, 1.319352626800537, 0.16008469462394714, -0.5316332578659058, -0.36182132363319397, -0.1967848688364029, 0.3270508646965027, 1.2504925727844238, 1.6217334270477295, -0.525624692440033, -0.7808817028999329, -0.5636646151542664, 1.180812954902649, 1.449508786201477, -1.7508575916290283, 1.0402721166610718, 0.19013169407844543, 1.7505098581314087, 0.764631986618042, -0.683212399482727, -2.4598934650421143, -2.38751220703125, -1.3616503477096558, -0.6372201442718506, 0.37142324447631836, -0.46607673168182373, -1.4847652912139893, -0.6440221667289734, -0.4294356405735016, -0.1903567761182785, 0.15972764790058136, -1.429998517036438, 1.0950210094451904, -0.792147159576416, 0.34416407346725464, 0.9461296796798706, 2.3196513652801514, 1.1457792520523071, -0.8086794018745422, -1.0075079202651978, -0.08698921650648117, -2.6894829273223877, 0.9887017607688904, 0.8643349409103394, -0.24911941587924957, 1.5581567287445068, -0.4725114107131958, -1.508165955543518, -0.790422260761261, -0.10401647537946701, 0.27522575855255127, 0.6492378115653992, 1.1502989530563354, 0.5998895764350891, 1.128949522972107, 0.9122388958930969, -0.8182283043861389, 0.602496862411499, -0.8089860081672668, 0.1266598105430603, -0.7243276238441467, -0.746905505657196, 0.6495808959007263, 1.7317609786987305, -1.84099543094635, 0.7506780624389648, -1.2811013460159302, 1.2650820016860962, -1.0769394636154175, 1.3109036684036255, 1.1090420484542847, 1.5061299800872803, -0.24498622119426727, 2.959564208984375, -0.9963585734367371, -1.2725532054901123, -0.8915334343910217, 1.2974165678024292, -0.35934507846832275, -0.2673477232456207, 0.7627050876617432, -0.02731378562748432, 0.7895861864089966, 0.6668977737426758, 0.26203805208206177, -1.5967943668365479, 0.10512931644916534, 0.2757863700389862, 0.48039934039115906, 0.7321460843086243, 0.44298097491264343, -2.0053763389587402, 0.15844310820102692, -0.9255543947219849, 0.7417294979095459, -0.674312949180603, -0.02095046453177929, 0.976081907749176, -0.7148285508155823, -0.4532288610935211, -1.7522605657577515, -0.8647482395172119, -1.2632683515548706, -1.0447900295257568, -0.6030564904212952, -0.4017895460128784, 0.046891141682863235, 0.945070207118988, -0.1889394372701645, 0.8226713538169861, -1.0188943147659302, 1.1192783117294312, -2.087008237838745, -0.0935714915394783, -0.42589685320854187, 0.8518770933151245, -1.0391051769256592, 1.354436993598938, -0.6958050727844238, 0.02048688568174839, -0.7498694062232971, -1.1774072647094727, 0.6083211302757263, -0.09308049082756042, 2.5653133392333984, -0.13471387326717377, 3.1052603721618652, -0.9320502877235413, -0.5340566039085388, -1.9375483989715576, -0.8475695252418518, -1.19623863697052, 0.5715528726577759, -0.27854016423225403, -2.0267174243927, -1.0746872425079346, 0.47779184579849243, 1.649667501449585, -0.3573162853717804, -0.23302289843559265, -1.4051200151443481, -0.521921694278717, 0.86119145154953, -0.2502933144569397, -0.2650129497051239, -0.14327207207679749, -2.2497239112854004, -0.5491355061531067, -1.11988365650177, -0.23391607403755188, -1.4879635572433472, -0.9408263564109802, 1.4869011640548706, 0.9156480431556702, 1.352798342704773, 0.33119997382164, 0.4148455262184143, -0.17132030427455902, -1.1982340812683105, 1.6800302267074585, 1.2878390550613403, -0.15626055002212524, 0.7445055842399597, -0.29791897535324097, -0.28901493549346924, -1.2405338287353516, 0.03880704566836357, -1.157086968421936, 0.5204305052757263, -0.09946674108505249, 0.7844212651252747, 0.5297186970710754, 0.7304678559303284, 0.44241002202033997, -0.029527541249990463, -1.0114784240722656, 0.1914610117673874, -0.33585453033447266, 1.3181122541427612, 0.3567711412906647, 0.36733192205429077, -0.9433515071868896, 1.2652897834777832, -0.6770933270454407, 0.8688731789588928, -1.2978756427764893, -0.5215722322463989, 0.5547640323638916, 1.907407522201538, 1.2776830196380615, 0.7494766712188721, -0.9246189594268799, -0.785764217376709, -0.12155355513095856, -0.8219185471534729, -1.0311143398284912, -0.8101656436920166, -2.600780963897705, -0.2122170478105545, -1.0133278369903564, 0.3313552439212799, 0.48230037093162537, -1.2837265729904175, -0.18682055175304413, 0.9320133924484253, 0.5198205709457397, 1.1735734939575195, -0.07341544330120087, 0.3897348642349243, -0.11900878697633743, 0.14979611337184906, 0.04256761074066162, 0.19044704735279083, -0.7965036034584045, -0.7094820141792297, -0.198092982172966, -0.7162916660308838, -0.21227172017097473, -1.2250781059265137, -1.1742079257965088, 0.04386306181550026, 0.81339031457901, -2.9261884689331055, 0.2649073004722595, -0.7528672814369202, -2.576977252960205, -1.488013744354248, 0.1551988124847412, 0.5256516337394714, 0.02187853492796421, 0.15665477514266968, -0.9064615964889526, -0.481282114982605, -0.8120900988578796, -0.6417906880378723, -0.9762998223304749, 0.9048706889152527, -1.1180591583251953, 0.8006857633590698, 0.9323644042015076, 2.71280837059021, 1.2693564891815186, 0.3402509093284607, 0.5109425187110901, 0.08119550347328186, 1.1951733827590942, -0.20316068828105927, 1.1077258586883545, -1.3978064060211182, -0.05813862755894661, 0.7730449438095093, -1.4715286493301392, 0.9377687573432922, 0.6514263153076172, 0.25699830055236816, 0.23488876223564148, -1.0936390161514282, -1.8811064958572388, -0.42055776715278625, -0.3973628878593445, -0.18247748911380768, 0.13764823973178864, 1.2341487407684326, -0.841546356678009, -1.2971529960632324, -2.127108335494995, 0.47815263271331787, 0.018053516745567322, -0.029291771352291107, -0.5858962535858154, 0.3170892596244812, -0.4152098298072815, 0.8976376056671143, 0.21735583245754242, 0.021094413474202156, -0.6042321920394897, -0.39277443289756775, 0.0365944504737854, 2.4417271614074707, -0.3986082971096039, 1.0205141305923462, -2.1801862716674805, -0.34610238671302795, -0.5645467042922974, 1.5702965259552002, 1.243586540222168, -0.6076926589012146, -0.9220029711723328, 1.058211326599121, 0.3627243936061859, -0.7556823492050171, -1.4063838720321655, -0.07873788475990295, -0.29844963550567627, 0.1700076162815094, -0.18194231390953064, -1.3272508382797241, 1.4058905839920044, 1.3809096813201904, 0.47481027245521545, 1.0544934272766113, -2.7828705310821533, 0.15830373764038086, -2.1125383377075195, -0.09239199757575989, 0.34225550293922424, -0.10905718803405762, -0.9242133498191833, 1.1180020570755005, -1.1171445846557617, -1.4698917865753174, -0.40392404794692993, -1.0585956573486328, -1.8457562923431396, 1.0045766830444336, -0.7367802858352661, -0.0846225693821907, 0.17665772140026093, 0.7503447532653809, -1.3252023458480835, -0.3743496537208557, -1.24569833278656, 1.612746238708496, -0.33385175466537476, -0.582166850566864, -0.5509014129638672, -0.08493493497371674, -0.5331461429595947, -0.32115647196769714, 0.37114280462265015, -0.3446204662322998, 0.07616278529167175, 0.9135767221450806, -0.8557145595550537, -0.8994538187980652, 0.37374967336654663, 2.102881908416748, 1.548092007637024, -0.5919224619865417, -1.0558300018310547, 0.6129322648048401, -0.8573818802833557, 0.1585158407688141, 1.2436223030090332, -0.8447571992874146, -1.2324670553207397, 0.6597755551338196, 0.7243028283119202, 0.10533119738101959, 1.2088685035705566, -1.2899531126022339, 0.6052082777023315, -0.024909939616918564, -1.1319928169250488, 1.4686005115509033, -1.0816730260849, -0.47257137298583984, -0.2574792504310608, 0.432330459356308, 1.9372411966323853, 2.3131797313690186, 0.7866550087928772, -0.15462929010391235, -1.4565818309783936, 0.5353497266769409, 0.697373628616333, -0.712705135345459, 0.23824141919612885, 0.41786909103393555, -1.5327309370040894, -0.007947469130158424, -0.3271055519580841, 2.3708770275115967, 0.5581466555595398, 0.2417447715997696, 0.4804055690765381, -0.10770435631275177, -0.3363596796989441, -1.0734672546386719, -0.7655932903289795, -2.6189591884613037, 0.6303543448448181, 1.1563637256622314, 0.8576611280441284, 0.014605565927922726, -0.08476512134075165, -0.9173275232315063, -0.011268348433077335, 0.446056991815567, 0.28895631432533264, -0.3601560890674591, -0.9570170044898987, -1.1175358295440674, 0.2923327088356018, -0.3260377049446106, -0.6222869157791138, -1.5869840383529663, -0.3619510531425476, -0.666324257850647, -0.7151808142662048, -0.7124887704849243, 0.24298764765262604, 0.03206472471356392, -0.6338837742805481, -1.218555212020874, -0.9906500577926636, -0.09705904126167297, 0.5575168132781982, 1.237356424331665, 0.8944684267044067, -0.2603069543838501, 0.782136857509613, 1.133139967918396, -1.0333322286605835, -1.6686577796936035, 1.2036429643630981, -0.5495538115501404, 0.8189043402671814, 1.1504205465316772, -0.2150418609380722, -1.3810099363327026, 3.1035678386688232, -0.5484414100646973, 0.2626040577888489, -0.4734361469745636, 0.7680450677871704, -0.6275657415390015, 0.49743926525115967, 0.610053300857544, 0.3803490996360779], [-1.1641120910644531, -0.3969089090824127, -1.1883342266082764, -2.3497583866119385, -1.611266016960144, 0.8690343499183655, -0.8312504291534424, -0.008238195441663265, -1.1644326448440552, 0.32457435131073, -0.35946303606033325, -0.33421963453292847, -0.5446890592575073, 1.9743096828460693, 1.8949824571609497, 0.2084425836801529, -0.5243620872497559, 0.9283992648124695, 0.8582333326339722, 1.1066606044769287, -0.03815734386444092, -1.0312626361846924, 0.8992280960083008, -1.2995946407318115, -0.5700811743736267, 0.9624509811401367, 0.3902983069419861, 0.5220616459846497, 1.1645534038543701, 1.3722896575927734, 0.5181922316551208, 1.159706473350525, -0.693131148815155, -0.30979278683662415, -1.1492844820022583, -1.0720012187957764, 1.3576459884643555, 0.6849204301834106, -0.27169787883758545, 0.7683672308921814, -0.7615012526512146, 0.8612222671508789, -0.6988521814346313, -1.170371413230896, -1.048162579536438, -1.3371591567993164, -0.03471541777253151, -0.41202476620674133, 1.3539162874221802, 1.1944196224212646, -0.23920458555221558, -0.6014357805252075, -1.121772050857544, 1.0630937814712524, -0.8328173160552979, -0.19612641632556915, -0.8079542517662048, 1.0272566080093384, 0.9019403457641602, 0.3667338490486145, 0.3060241639614105, 0.21735836565494537, 0.9111028909683228, 1.1846474409103394, 0.9810697436332703, -1.705763339996338, -0.8744531869888306, 1.40240478515625, 0.9003519415855408, 1.1654542684555054, 0.8853369951248169, 0.044073596596717834, -1.244317889213562, 1.1095104217529297, -1.8454147577285767, 1.4122998714447021, -0.8242314457893372, 0.14315563440322876, -2.196331739425659, -0.5533048510551453, -0.6730343699455261, 1.023525595664978, -0.8807294964790344, 0.2583465874195099, -0.38261479139328003, -0.13158690929412842, 0.4527953267097473, -0.247569739818573, -1.094425082206726, -1.6637170314788818, 1.7497588396072388, -0.8124097585678101, -0.10936258733272552, -0.34074363112449646, -0.522997260093689, -2.148202896118164, -0.18649129569530487, 0.8458476066589355, 1.6102501153945923, -0.15929287672042847, 1.7649426460266113, 0.5773364901542664, 1.9919003248214722, -0.0760495662689209, -0.7423555850982666, -0.4105125367641449, 0.007315150462090969, 0.14122232794761658, 0.701516330242157, -0.2930464744567871, 0.677449643611908, -0.6484549641609192, -0.2827964425086975, 0.7001593112945557, -2.2879302501678467, -0.4875858426094055, -0.4398009777069092, -0.8090290427207947, 0.7277413606643677, 1.7100309133529663, -1.8441492319107056, -0.4766541123390198, 2.1596312522888184, -0.07851243019104004, 0.20049744844436646, 1.4466657638549805, -0.3540946841239929, 1.1881438493728638, -0.14403584599494934, 0.1323971450328827, -0.19280041754245758, 2.270158052444458, -0.7782729268074036, 0.3202517032623291, -0.2958093285560608, 1.8837835788726807, -0.8548331260681152, -0.09856649488210678, -0.5467389822006226, -0.4585224390029907, -0.7600008845329285, -0.6830580234527588, -0.04749453812837601, 0.16718073189258575, -0.3345660865306854, 0.18453139066696167, -0.08870519697666168, 0.6924297213554382, 0.4912523627281189, 0.6046538352966309, 1.2301970720291138, -0.5918862819671631, -0.07288683950901031, 0.6227616667747498, -0.9449819326400757, 0.07940391451120377, 0.6430226564407349, -2.8854422569274902, 1.1304818391799927, -0.7740575671195984, 0.23311640322208405, 0.568731427192688, 0.7465773820877075, -1.343741774559021, 0.4162284731864929, -1.2072242498397827, -1.4926068782806396, -1.4304752349853516, -1.840946912765503, 0.19699421525001526, 0.26053088903427124, -0.1821308135986328, 0.8578755855560303, 1.7316854000091553, -0.8695284724235535, 1.7398569583892822, -0.6382884979248047, -0.7122294902801514, 0.3812461197376251, 1.281409740447998, -0.2292899191379547, 0.1253635287284851, 2.082496404647827, 0.8005775809288025, 0.1995851993560791, -0.8001318573951721, -1.405491590499878, -0.46117138862609863, -1.0411808490753174, 1.6961406469345093, -0.49590298533439636, -0.072604238986969, 1.5652897357940674, 0.490829199552536, 0.7779805064201355, -0.6223993897438049, 1.0848519802093506, -0.4024832248687744, -2.403520345687866, -1.430422306060791, 0.08887051790952682, -0.8136553764343262, -0.873374342918396, 0.7483708262443542, 0.08734521269798279, 0.05901281163096428, 0.4933658838272095, -1.4877269268035889, 0.20183178782463074, 0.1390455961227417, 0.2638014554977417, 0.8910996913909912, -0.4662635624408722, -1.4957619905471802, 0.39946648478507996, -0.47391194105148315, -0.6628214716911316, 1.009200096130371, -0.7162541151046753, -0.24517785012722015, 0.9888204336166382, -1.6733293533325195, 0.8993068933486938, 0.2622026205062866, 0.4857213497161865, 1.1865193843841553, -0.7420729994773865, -0.6314541101455688, -0.6807661056518555, 0.511546790599823, -0.9576524496078491, -0.6385275721549988, 0.6593495607376099, -2.2464962005615234, -2.211181163787842, 0.6467936038970947, 0.10264509916305542, -0.12077774852514267, 0.4380599558353424, 0.965269148349762, 0.5390127897262573, 0.0599457211792469, 1.037790060043335, 0.10333584249019623, -0.5448898673057556, -0.9387337565422058, 0.48282963037490845, -0.9019659757614136, -1.8965418338775635, 0.06923986226320267, -0.015433663502335548, 1.516252040863037, -1.651923656463623, -0.015450081788003445, 0.6249148845672607, -0.7433260679244995, 0.6293718218803406, 1.1179002523422241, 1.2288750410079956, 0.3033965826034546, 1.0263710021972656, 0.022417353466153145, 0.9401997327804565, 0.6312059164047241, -0.5404523611068726, 1.3650271892547607, -1.0167200565338135, 0.9868568778038025, -1.4533659219741821, -1.1878429651260376, -1.5661548376083374, -2.04896879196167, 0.6099304556846619, -0.37030208110809326, -0.5136938095092773, 2.1704063415527344, 0.5800874829292297, 0.8016207814216614, 1.984114646911621, -0.22371722757816315, -1.3347535133361816, 0.7743260264396667, 0.3136395812034607, 1.4225094318389893, -0.2657052278518677, 0.5984861850738525, 0.9015564918518066, 1.29252028465271, -0.6004464626312256, -0.03506045415997505, -0.8315197825431824, 0.7146776914596558, 0.3184763193130493, 1.1999735832214355, -0.017325423657894135, -0.03033273108303547, 0.5912054181098938, 0.1450735330581665, 0.666002631187439, -1.0590001344680786, 0.10401010513305664, -1.3882973194122314, -1.120970606803894, -0.3197769522666931, -1.6606014966964722, -1.3556573390960693, -1.1953238248825073, -1.1521799564361572, 1.3599095344543457, -0.8413224816322327, 0.40947070717811584, -1.4890793561935425, 2.2851333618164062, 1.1687380075454712, -0.4634215533733368, -0.8133190274238586, 0.38405290246009827, -1.8943403959274292, -0.8875321745872498, 0.15692128241062164, -0.9321100115776062, 0.40968236327171326, -0.6360268592834473, -0.13811536133289337, 0.6188097596168518, -1.8142428398132324, 0.8478901386260986, 0.8151818513870239, -0.06589720398187637, -1.0101677179336548, 1.2799532413482666, -0.05864999070763588, -0.3267017900943756, -0.36619457602500916, -0.8518833518028259, 0.3353012502193451, 0.2805192470550537, 1.0042471885681152, 2.1310813426971436, -0.2128666639328003, 0.4882635176181793, -2.5234591960906982, -0.9550268650054932, -0.04229382053017616, 0.9986363649368286, 0.6348395347595215, -0.8365850448608398, 0.5202306509017944, -1.2264280319213867, -0.511720597743988, -0.290349543094635, 0.8357152342796326, 1.674606442451477, -0.27012741565704346, 0.8288801908493042, -0.3065668046474457, 0.4410603642463684, -1.0193829536437988, -0.21527183055877686, -2.0194222927093506, 1.5096490383148193, 1.9595221281051636, -1.3684617280960083, -0.40835368633270264, -1.592360019683838, -0.49456122517585754, -0.8625523447990417, 0.13092094659805298, -0.8889329433441162, 2.605700731277466, 0.1053939163684845, -0.7007243633270264, 1.4376144409179688, -0.17405642569065094, -1.5945249795913696, 0.5423160791397095, 2.117668390274048, 0.750447154045105, -1.0470572710037231, -1.1612040996551514, -0.9271770715713501, -0.05771893635392189, 1.6506257057189941, 0.6064151525497437, -1.5888339281082153, -0.8362584114074707, -0.9446710348129272, 0.6887989640235901, 0.19205395877361298, -1.1085678339004517, -0.9695850610733032, 1.15230393409729, 0.7510984539985657, 0.4326910972595215, 1.0376988649368286, 1.4414920806884766, 1.0986108779907227, -1.055994987487793, -0.09543764591217041, -0.41728562116622925, -1.2795755863189697, -1.8041486740112305, -1.2376388311386108, 0.868000864982605, 0.061030689626932144, -0.8286911249160767, -1.377419352531433, 0.5130761861801147, 1.8627164363861084, 0.14232678711414337, 1.8180959224700928, 1.0368229150772095, 0.6249648332595825, 0.928945004940033, -0.48834967613220215, -0.26803115010261536, 0.24076338112354279, -0.556246280670166, -0.07145130634307861, -0.9449045062065125, 2.5103461742401123, 1.4106522798538208, -0.7799676656723022, 0.9495749473571777, -0.6509125828742981, -0.377097487449646, -0.5926190614700317, -1.2454253435134888, -1.850149393081665, -0.27331840991973877, 0.04633130505681038, -0.07972954213619232, -1.2235456705093384, -1.416533350944519, -0.7091886401176453, 0.7643150687217712, -0.394881010055542, -0.6404888033866882, -0.3498361110687256, -2.1770825386047363, 0.37640002369880676, -1.620090126991272, -0.6620094180107117, -0.8115564584732056, -0.21376535296440125, -0.6928312182426453, 0.5114571452140808, 0.8551399111747742, 1.1532624959945679, 1.5552873611450195, 0.5509639978408813, 0.5246170163154602, 0.39168208837509155, 1.1092535257339478, 1.1763184070587158, -0.7833226919174194, 0.21367542445659637, 1.7834510803222656, -1.4123730659484863, 0.45418235659599304, 0.49678975343704224, 0.133707195520401, 1.2992396354675293, 0.2983768880367279, 1.1204931735992432, -1.848496913909912, -1.5925633907318115, 0.9839953780174255, -0.24944517016410828, -1.2355048656463623, -0.508886992931366, -1.058349370956421, -0.7336030602455139, -1.4392956495285034, -0.32316192984580994, 0.09907501935958862, -0.04993477463722229, 0.029524443671107292, -0.4796169698238373, 1.048833966255188, 0.3014066815376282, 0.3178335130214691, 1.3339571952819824, -0.07727212458848953, -0.004439878277480602, -0.1434544175863266, 0.7643104791641235, 1.4877996444702148, -0.6463485360145569, -1.7873367071151733, 1.7695478200912476, -0.7857427000999451, 0.8906004428863525, 0.2331652194261551, 0.36277642846107483, 0.2806571424007416, -1.1849472522735596, -0.3763657212257385, -0.3197104334831238, 1.0609034299850464, 0.6168116927146912, 1.3660660982131958, -0.2656400501728058, 0.2365618646144867, -0.23270593583583832, 1.1719671487808228, 2.155529022216797, 0.6006574034690857, 0.5108769536018372, -1.7268106937408447, 1.4429723024368286, -0.19371356070041656, -1.0105066299438477, 0.06426336616277695, -0.657233476638794, 0.4941701591014862, 0.17557302117347717, 0.004503628239035606, 2.8912770748138428, -0.4009039103984833, 1.101047158241272, 1.2851364612579346, 1.310716986656189, -0.41218918561935425, -1.5399785041809082, -2.5957272052764893, -0.504368245601654, 0.5930175185203552, -0.5875793099403381, 0.9569105505943298, 2.424692153930664, 0.8135508894920349, -0.012857750989496708, 1.4854384660720825, 1.0258913040161133, 1.355006217956543, 1.5393744707107544, -1.2454984188079834, 1.4496110677719116, -0.2741931080818176, -0.11836516112089157, 1.2424826622009277, 1.1875066757202148, 0.4065346419811249, -1.3122937679290771, 1.5559228658676147, 0.1745893657207489, 0.8180602788925171, -0.33469358086586, -0.23258180916309357, -0.22028350830078125, 0.9997540712356567, -1.3753795623779297, -0.054350513964891434, -0.57346510887146, -1.0435079336166382, 1.1707332134246826, 0.031104078516364098, -0.0930875614285469, 1.0037662982940674, 1.508671760559082, 0.5914170742034912, 2.3398802280426025, -0.08803890645503998, 0.18125906586647034, -0.21143190562725067, 0.542618989944458, -1.0665181875228882, -1.4450031518936157, 0.9700129628181458, 0.09582651406526566, -0.8126970529556274, 1.0815736055374146, 1.355224370956421, -1.1417491436004639, 0.7155461311340332, 1.1034440994262695, 0.6071893572807312, 1.5305588245391846, 3.215959310531616, 0.07344327867031097, -1.366801381111145, -0.8823007345199585, -0.48363831639289856, -2.193303108215332, -1.2836164236068726, -0.25769609212875366, 0.34452807903289795, -0.6690048575401306, 0.894167423248291, -0.9119674563407898, 0.6727211475372314, -0.5057582259178162, 0.416358083486557, 1.3440243005752563, 0.9717559218406677, -0.8452897667884827, -0.11893623322248459, -0.010139894671738148, 1.4952093362808228, 0.578406035900116, 1.122067928314209, 0.385152667760849, -1.5308219194412231, -0.002440808340907097, -1.527081847190857, 2.7512547969818115, 0.14282843470573425, -1.1527042388916016, 0.3479746878147125, 0.31021344661712646, 0.32446062564849854, 0.3171827793121338, 0.9768832325935364, -1.6541560888290405, 1.1519863605499268, 0.37429967522621155, 0.8277053236961365, -0.6720991730690002, -0.4937228560447693, 0.12019363790750504, -0.19579531252384186, -1.795778751373291, 1.450315237045288, 1.5667084455490112, 0.2525028586387634, -0.5391145944595337, 1.2396297454833984, -0.5478067398071289, 0.9507506489753723, 2.040181875228882, 1.9452133178710938, -0.07763253152370453, -0.5727421045303345, -1.6908419132232666, 1.2129098176956177, 1.5710563659667969, 0.5453896522521973, -1.541002869606018, 2.087175130844116, 0.3918974697589874, 0.32021066546440125, -1.935154676437378, 1.038201928138733, 1.5306843519210815, -0.15274226665496826, 2.0267655849456787, 0.23098628222942352, -0.38976776599884033, 1.896462321281433, 0.581260621547699, -0.2410627156496048, 1.1617200374603271, -0.43539339303970337, 1.3485771417617798, -0.5225789546966553, 0.005802182946354151, -0.35203349590301514, -1.2860122919082642, 0.1641356647014618, -0.7675758600234985, -1.0561600923538208, 0.6412257552146912, -1.0788503885269165, 1.3187274932861328, 0.6726728081703186, -0.04896942153573036, -0.013621916063129902, -1.4599512815475464, -0.4408135414123535, 1.9644062519073486, 0.2036992609500885, -0.17343293130397797, 0.4337114095687866, -0.6882206797599792, -1.563712477684021, -0.40046727657318115, 0.08922674506902695, -0.34366315603256226, 0.6781996488571167, -1.1437121629714966, 1.06508207321167, 1.196420431137085, -0.38940826058387756, 0.6275305151939392, 0.9887714982032776, -0.36295631527900696, 0.6461260914802551, -1.4033457040786743, 0.7520384788513184, 0.364164799451828, -0.25274500250816345, 0.3427124619483948, -0.2931664288043976, 1.1639785766601562, -0.3077080249786377, 0.03702492639422417, 1.0497934818267822, 0.8164302706718445, 2.4374167919158936, -3.3248655796051025, 1.5611913204193115, -0.1321256160736084, 0.056666404008865356, 1.0226924419403076, 0.17202496528625488, 1.2286081314086914, -0.06443826854228973, 0.13392983376979828, -0.2611244022846222, -1.425933599472046, 0.4257926940917969, -0.6407708525657654, -0.9387255907058716, -1.7576782703399658, 0.07462006062269211, 0.08437814563512802, 1.8449947834014893, 0.9756403565406799, -1.1520066261291504, -0.1855962723493576, -0.1213751956820488, -1.634916067123413, 1.0412946939468384, 2.06008243560791, 0.3394547700881958, -0.038639962673187256, -0.4163302183151245, -1.300883412361145, 0.9096865653991699, -2.1642978191375732, 0.1770200878381729, -1.2140324115753174, -0.4533780515193939, -1.170285701751709, 0.7586944103240967, 0.7768293619155884, -0.306295782327652, -0.871063232421875, -1.8353383541107178, 1.5272459983825684, 0.2788918614387512, 1.1497269868850708, 1.2806886434555054, 1.427941918373108, -0.30891096591949463, -0.642873227596283, 0.49376532435417175, 0.29746636748313904, 0.29355886578559875, 0.700437068939209, 1.1134216785430908, -0.6806609630584717, 0.3928002715110779, -0.7070856690406799, 1.1108342409133911, 0.5048828125, -2.192610740661621, -1.8141931295394897, -1.253444790840149, 0.0447021909058094, 1.5739067792892456, -0.7312699556350708, -0.9064878821372986, 0.4716149568557739, -0.5376332998275757, -0.5201663970947266, 1.7285220623016357, 0.6764577031135559, 1.0168179273605347, 0.12187656760215759, 1.1617109775543213, 0.6518608331680298, 0.42814841866493225, -0.5482814311981201, 0.5908330678939819, 1.0805085897445679, -0.1868065893650055, 1.9298208951950073, -1.919319987297058, 3.0587692260742188, -0.9658706188201904, -0.2952001690864563, 0.5085131525993347, 0.708915650844574, -1.0764883756637573, -1.1671438217163086, -0.32707834243774414, -0.9228200912475586, -1.9731554985046387, 1.9916672706604004, -0.5914479494094849, 0.05385648086667061, 0.8630463480949402, 0.7560186982154846, 0.4169504642486572, 0.8269687294960022, 1.0517429113388062, 0.9672116637229919, -1.17240571975708, 0.009376647882163525, -1.984278917312622, 0.04322696849703789, 0.9408422112464905, -1.3728879690170288, 0.7048395872116089, 1.677017092704773, 1.3561580181121826, -1.0819482803344727, -0.7692757844924927, -0.644873857498169, 0.501118540763855, -0.5765076279640198, -1.6928598880767822, -2.0041778087615967, -2.8470258712768555, 1.2998205423355103, -2.517180919647217, 1.0299235582351685, 0.016495229676365852, -0.31116101145744324, -0.08329148590564728, 0.49557194113731384, 0.4331866204738617, -1.361689567565918, -1.0081228017807007, 1.1675041913986206, 0.7180668711662292, 2.498457670211792, 0.17547771334648132, 0.8765491247177124, -0.7064706087112427, -2.162541151046753, 0.3381110429763794, -0.7278127074241638, 0.7269962430000305, 0.3693089187145233, -0.2787744402885437, -2.04742431640625, -0.17763647437095642, -2.3227927684783936, 0.014629149809479713, -0.09199479967355728, 1.8137539625167847, 0.4290613532066345, -0.876726508140564, -0.05292452871799469, 0.5894220471382141, -0.03922775387763977, -0.30130380392074585, -0.7540006041526794, 0.2064833790063858, -0.5187020301818848, -1.0932540893554688, -0.1261925846338272, -0.5150795578956604, 0.8359863758087158, -1.380937933921814, 0.49831050634384155, -0.3545575737953186, -0.34035724401474, -1.341155767440796, 0.4614256024360657, -0.6390987038612366, 1.060433268547058, 1.1972920894622803, 0.990172266960144, 0.023342672735452652, 0.2071327269077301, 0.7910577654838562, -1.2773325443267822, 1.5495984554290771, -0.26311084628105164, 1.519059419631958, 0.5011513829231262, 0.05247892811894417, -0.5181642770767212, 1.9008883237838745, -0.4381016194820404, -1.2971431016921997, -0.9710901379585266, -0.5416361093521118, 2.2462480068206787, 0.07162129878997803, -1.0548152923583984, -1.4247064590454102, -2.522406816482544, 0.06295981258153915, -0.16183988749980927, -0.3523063063621521, -0.02414518967270851, -0.8992500901222229, 0.42573288083076477, -2.7381129264831543, 1.4512343406677246, -1.9617449045181274, 0.5561217665672302, 0.12929902970790863, 0.3840200901031494, -0.14863300323486328, -0.2801900804042816, -0.8020613789558411, 0.6958366632461548, -0.6522722840309143, -1.7845882177352905, 0.8233245015144348, 1.0703763961791992, -3.1979012489318848, 0.2685810625553131, 0.45592567324638367, -0.6993114352226257, -0.2718280851840973, 0.2123211771249771, -1.6560924053192139, 0.5427345037460327, -2.9634392261505127, -2.4290220737457275, 0.784719705581665, -0.020791208371520042, 0.08055006712675095, 0.41035860776901245, 0.6155051589012146, 0.13277976214885712, -0.8219906687736511, -0.4239904582500458, 0.06615376472473145, 0.47025391459465027, -1.5901867151260376, 0.2559059262275696, -0.06614987552165985, -0.49341097474098206, -0.6053796410560608, 0.2886357605457306, 1.0878169536590576, -0.15548744797706604, -1.0443575382232666, -0.13920044898986816, -1.8798710107803345, -1.556002140045166, 1.736402153968811, -0.04344719275832176, 0.020266342908143997, 0.7793504595756531, -0.5338797569274902, -0.8738304376602173, 0.09385466575622559, -0.5095292329788208, -0.6013039946556091, 0.6212668418884277, 0.527553915977478, 0.4095441997051239, -0.8364208340644836, -0.6852431297302246, -0.18574319779872894, 1.8096539974212646, -0.6639056205749512, 0.6573107242584229, 0.2158776819705963, 0.8000982999801636, 1.063919186592102, 0.737422525882721, 2.0412020683288574, 1.2498500347137451, -1.3929390907287598, -0.3083430230617523, -0.45140403509140015, -1.7201929092407227, 0.07427993416786194, -1.0752058029174805, -0.7272034287452698, -0.7246419787406921, -2.0065741539001465, 0.21766874194145203, 1.3761324882507324, 1.0822434425354004, -1.0502582788467407, 0.3453483283519745, -1.1719340085983276, -0.0835726261138916, 0.2840985357761383, 0.28901585936546326, 1.651547908782959, 1.4340497255325317, 1.1542657613754272, -1.8691797256469727, 1.0833534002304077, 0.22049714624881744, 0.953198254108429, -1.442177176475525, -0.6935399174690247, -0.5258642435073853, -1.5400876998901367, 0.6077755689620972, -1.1612296104431152, 0.8850347399711609, 0.2910436987876892, 0.11742180585861206, -0.466967910528183, 0.13282527029514313, -0.5380618572235107], [0.328620582818985, 1.1262187957763672, -0.16665156185626984, 0.4060732126235962, 0.8724951148033142, 0.5783786177635193, 0.7914848327636719, 1.8043345212936401, -2.0013599395751953, 1.196365237236023, 0.11733631044626236, 0.6370500326156616, -0.6835517883300781, -1.0577410459518433, -0.46968355774879456, -1.1117600202560425, 0.35103777050971985, -2.1230897903442383, 0.11745771020650864, -1.0720672607421875, 0.6175081729888916, -1.473895788192749, 0.32689663767814636, 0.7682324647903442, -0.9447293877601624, -1.709098219871521, -0.7338622808456421, 0.2532491087913513, -1.1815911531448364, -1.2797319889068604, 0.6768861413002014, 0.26825565099716187, -0.056379348039627075, -0.8215741515159607, 0.42261767387390137, -0.8536680340766907, 2.4790701866149902, -1.1790202856063843, 0.39839109778404236, -0.6487302780151367, -0.791374683380127, -0.37553247809410095, -0.16742928326129913, 0.12493763118982315, -1.021946907043457, -1.029022216796875, 0.7239935994148254, 2.063227415084839, 1.0308613777160645, 0.1330147385597229, -0.38053539395332336, -1.1260359287261963, -0.43474462628364563, -0.9208627939224243, -0.4478299021720886, 0.5208923816680908, -0.1666685789823532, 0.045443229377269745, -0.8655115962028503, 0.3687862753868103, 0.15121278166770935, -0.8686646819114685, -0.7512343525886536, -0.480097234249115, -0.601371705532074, -0.6329839825630188, -0.9901520609855652, 0.9290540218353271, 0.37683016061782837, -0.4499557614326477, -0.20764397084712982, 1.0976463556289673, 0.35498616099357605, -1.1391178369522095, -0.07397425174713135, 0.8018823862075806, -0.010720552876591682, 0.2705559730529785, 1.1647285223007202, -0.8662434816360474, 1.5752959251403809, 0.5020949840545654, 1.1208511590957642, -0.7366954684257507, 0.3558214604854584, 0.5635177493095398, 1.7994508743286133, -0.9272558689117432, -2.740607976913452, 1.1422817707061768, -0.40193819999694824, 1.2766903638839722, -1.4081238508224487, -0.8505874872207642, 0.4245617985725403, -0.9786903262138367, 0.9657420516014099, 0.563917875289917, -0.5406448245048523, -0.18294726312160492, -0.047146987169981, -0.3762068748474121, 0.1535797417163849, -0.8198685050010681, 0.7446469068527222, 1.6465433835983276, -1.4112383127212524, -6.501204916276038e-05, -0.11886629462242126, 2.15545392036438, 1.291445016860962, -0.40078121423721313, 1.5961583852767944, 0.004157414194196463, -0.03340468555688858, 0.21589751541614532, 0.6398479342460632, -0.6550628542900085, -1.8640551567077637, 0.7988986968994141, -0.6892556548118591, 1.9263235330581665, -1.276771068572998, 0.7925539612770081, 0.2724400460720062, 0.18731839954853058, 0.14130710065364838, -0.14047162234783173, 0.13837440311908722, -0.379552960395813, -0.5324285626411438, -0.5072619915008545, 0.811676561832428, 0.2862972319126129, -0.1919042319059372, -0.6516860723495483, -0.4631688892841339, 1.0494147539138794, -0.2708650231361389, 0.8321598768234253, 3.1799325942993164, -1.3534431457519531, 1.1094800233840942, -1.3015486001968384, 1.4033538103103638, -0.42827585339546204, -0.7770565152168274, -0.5900310277938843, -1.8161511421203613, -0.7561963200569153, -2.240934133529663, -0.2567133605480194, 0.6548356413841248, -0.821133553981781, -0.9321056008338928, 0.7757388353347778, 0.040672581642866135, 0.41648411750793457, -0.1040678396821022, 1.1866563558578491, 0.08043690770864487, 0.5444843769073486, 0.5115967988967896, 0.5720095634460449, 0.3519023358821869, 0.18636032938957214, 0.5082873702049255, -0.2503696382045746, 0.6814813017845154, 0.1087176501750946, 2.409313678741455, -0.5532045960426331, 0.5791193842887878, -0.24459302425384521, 0.44443652033805847, 1.3490585088729858, 0.35463565587997437, -1.416250467300415, 0.7216426730155945, -1.3307201862335205, -0.879442572593689, -1.6161317825317383, -1.2811416387557983, 0.9712274074554443, -1.517578125, 0.6794564127922058, 0.1800963282585144, 0.6134698987007141, -0.5291522741317749, -0.4535863995552063, 0.3387172222137451, 1.0611629486083984, 0.8616043925285339, -0.24727456271648407, 1.100249171257019, 2.068737745285034, 1.5004292726516724, 0.34742358326911926, 0.5963472127914429, -0.3788105547428131, -0.17059046030044556, -1.342962622642517, 1.1703165769577026, 0.32888785004615784, -0.26976633071899414, -0.31317493319511414, 0.31656914949417114, -0.8208540678024292, -0.3448110818862915, 0.052261222153902054, 1.8869582414627075, -0.9736945033073425, -0.31930026412010193, 1.107003092765808, 2.0689589977264404, 1.344688057899475, 0.8121373653411865, -0.0010538334026932716, -0.10428620129823685, -0.8561356663703918, 1.0445291996002197, 0.5176385045051575, 0.7994566559791565, -0.08763691037893295, 0.0207215528935194, -0.5278851389884949, -0.35640278458595276, 0.7515433430671692, -1.7086492776870728, -0.5250983834266663, -1.6590380668640137, -0.4293559789657593, 1.2146857976913452, 0.6014537215232849, 0.5127989053726196, -0.13177335262298584, -0.1476820856332779, 0.44601795077323914, 0.4696958661079407, 0.7723305225372314, 1.2795445919036865, -2.0729472637176514, -0.7627959847450256, -0.39986705780029297, -0.020746985450387, -0.9179854393005371, 0.07658509910106659, 0.3082068860530853, -0.9191382527351379, -0.5476077198982239, -0.22722291946411133, 0.3032059073448181, -1.6958202123641968, -0.9794674515724182, -0.8981302976608276, -1.7362576723098755, -0.5106009840965271, -0.24546660482883453, -1.0024651288986206, -1.233960747718811, -0.43447649478912354, -0.4617805778980255, -0.9181814193725586, 0.2616676688194275, 0.6400147676467896, 0.6935412883758545, -0.5048832297325134, 1.3433095216751099, 0.1377100795507431, 0.3502509295940399, 0.2798733115196228, -0.4581717252731323, 0.23235823214054108, -1.2102569341659546, -0.5294305682182312, 0.06379345059394836, 1.5215728282928467, -0.06891230493783951, 0.8930854201316833, -0.8609851002693176, 0.06393411010503769, 0.21468107402324677, -1.4501067399978638, 0.501715362071991, -1.7611879110336304, -2.547095775604248, 1.5268045663833618, -0.8981227278709412, -0.9811661839485168, 0.4075445234775543, 2.030172109603882, 0.09079813957214355, 0.5194119215011597, 0.1015746146440506, -0.6589378118515015, 0.16926294565200806, -1.1826188564300537, -0.8260392546653748, 0.011014405637979507, 1.6472145318984985, -0.3913077712059021, 0.8646438121795654, 0.837719738483429, -1.3012027740478516, -1.0766743421554565, -1.1301110982894897, 1.3115304708480835, 0.2912774085998535, -0.9434278607368469, 0.8564701676368713, -0.8631431460380554, 1.0036321878433228, 0.1385994404554367, 0.2608402371406555, -0.07416584342718124, -0.39815011620521545, 0.7985548973083496, -0.09984206408262253, -1.6348694562911987, 0.5510304570198059, -2.1261186599731445, -2.475578784942627, -0.5514132380485535, -1.2480111122131348, -0.44992727041244507, -0.3157641887664795, -0.009881202131509781, -0.8920581340789795, 1.4428153038024902, -2.2622416019439697, 1.1936578750610352, -0.5567309856414795, -0.347298264503479, 0.9281747937202454, -0.13467881083488464, -0.7586693167686462, -0.049514878541231155, 1.5156934261322021, 0.6037645936012268, -0.834606409072876, 0.062406640499830246, 0.7262446284294128, 1.525248408317566, -0.7006291151046753, 0.3689265549182892, -0.3018454611301422, 0.5718473196029663, -0.4966556429862976, 0.575934886932373, 1.2308039665222168, 2.20302414894104, 1.6017911434173584, 0.9472029209136963, -0.32610490918159485, -0.7769110798835754, -0.1322060078382492, 0.014054983854293823, -0.3707086145877838, -0.9972715973854065, -1.0138133764266968, 0.7966355085372925, 1.221218466758728, -1.172121286392212, -0.6206205487251282, -1.1748239994049072, 1.3111952543258667, -0.12638847529888153, 0.9799155592918396, 1.0299172401428223, -0.8490996956825256, 1.4114164113998413, 0.8872531056404114, 0.333967924118042, -0.7480714321136475, 1.6113260984420776, 0.7459124326705933, 1.8959027528762817, -1.3799338340759277, -0.4691944420337677, 0.8085874319076538, -0.9640553593635559, 0.8270590305328369, 0.5323801040649414, 0.08308281749486923, -0.4088824689388275, -0.8622428774833679, 0.799974799156189, -0.26413023471832275, -1.3295174837112427, -0.4736022651195526, -0.805376410484314, -0.7567440271377563, -0.6045808792114258, 0.1481017768383026, -1.3626620769500732, -0.06808505207300186, -0.07912787795066833, -1.4474551677703857, -0.11016768217086792, 1.4339264631271362, 0.5708624720573425, 1.299837350845337, 1.2511603832244873, 0.19418220221996307, 0.5114097595214844, 1.3680685758590698, -0.3917345404624939, -0.47883689403533936, 0.3560909926891327, 0.31938615441322327, -2.0002236366271973, 0.12656795978546143, 1.9533802270889282, -1.0110869407653809, -0.1747143417596817, -1.484090805053711, 0.9805435538291931, -0.18547706305980682, -1.3298832178115845, 0.08724205195903778, -0.8559548854827881, 1.2239127159118652, -0.9096139669418335, -0.00497426837682724, 0.12496955692768097, -1.126293420791626, -2.3020927906036377, -0.9648492932319641, 0.7835515737533569, 2.356902837753296, -0.12869790196418762, 1.614192247390747, 0.22019587457180023, 1.427476167678833, 1.2562956809997559, -0.6487465500831604, 1.07023024559021, 0.938358724117279, -1.385537028312683, 1.2000056505203247, -0.42436057329177856, -1.2665324211120605, -0.11402186751365662, 0.3780381977558136, -1.6114929914474487, 0.2092200219631195, -1.4503241777420044, -1.3604052066802979, -0.9521031379699707, -0.4696941077709198, 1.2037907838821411, -1.3527097702026367, -2.246032476425171, -0.42151808738708496, -1.4093221426010132, -1.030367374420166, 0.10803379863500595, -0.30682075023651123, -0.007415567059069872, 0.26516637206077576, -0.2637404203414917, -1.1107131242752075, 0.06874794512987137, 1.3674248456954956, -0.4820821285247803, -0.07876355201005936, -0.8668299913406372, 0.4276513159275055, 0.3918478488922119, -0.744817316532135, -1.4838913679122925, -0.8248240947723389, 0.38814783096313477, 1.3142002820968628, -0.5571244955062866, 0.23470546305179596, 1.786375641822815, 0.14771483838558197, 0.8736638426780701, 0.47825056314468384, -0.8403207659721375, 1.0771737098693848, 1.0603868961334229, 0.20112162828445435, -0.0808240994811058, 0.6162853240966797, 0.5704194903373718, 0.23547779023647308, -0.6026878952980042, 1.0318260192871094, 0.6205716729164124, 0.3966720998287201, 0.0732608214020729, 0.2633831799030304, 0.05830850079655647, 0.5469021797180176, 0.5065319538116455, -1.3192992210388184, 0.03013124316930771, 0.6910872459411621, 0.7945799231529236, -2.021390199661255, 0.9256994128227234, 0.0002108213520841673, 1.5509570837020874, 0.9927077889442444, 1.5429271459579468, -0.948165237903595, 0.80830979347229, 0.6237647533416748, 1.1753344535827637, 0.4280378222465515, 0.928867518901825, -1.1756243705749512, -0.4568774998188019, 0.5987688899040222, 0.8119748830795288, 0.42864376306533813, -1.8814111948013306, -1.5998319387435913, -0.7176708579063416, 1.0164847373962402, -0.9814361333847046, 1.2573095560073853, 1.5875909328460693, 0.3177388608455658, 0.6488023996353149, -0.5924589037895203, -0.2219248265028, 1.0621154308319092, 0.10732343047857285, 0.3142722249031067, -2.4828054904937744, 0.36202970147132874, 0.19210302829742432, -0.7966189384460449, 0.026147030293941498, 0.2502807378768921, -1.524309754371643, -1.3317469358444214, -0.03169839829206467, 0.2395242154598236, 1.3063031435012817, -1.6130911111831665, 2.4973702430725098, -0.5010840892791748, 1.795961618423462, -0.12514154613018036, -1.6808815002441406, -0.2316974699497223, 0.5151517391204834, -0.7640206217765808, 0.2228529453277588, 0.09529103338718414, -0.179830864071846, 0.9134041666984558, 0.4577103555202484, -0.48653584718704224, -0.22837936878204346, 0.4683188199996948, 0.10510766506195068, 0.6224253177642822, 1.4155547618865967, -1.5445724725723267, -0.17222292721271515, 0.6921839714050293, 0.7878269553184509, -0.7109000086784363, -1.6187975406646729, -0.7141802906990051, 0.5541507005691528, 1.3988909721374512, -1.8103575706481934, 0.5812814235687256, 0.9237278699874878, 0.6987648010253906, 0.6329077482223511, 0.6339824199676514, -0.8023801445960999, -0.5476801991462708, 0.9202532172203064, -0.11756034195423126, 0.6752423644065857, 0.6350976228713989, -0.027503473684191704, 0.9631075263023376, -1.478434443473816, -0.4531392753124237, -0.44445332884788513, -0.5255256295204163, 0.7009962797164917, 0.5654976963996887, -1.2474873065948486, 0.8545300364494324, -0.311407208442688, 1.3879796266555786, -1.2904354333877563, -0.7568174004554749, -0.22110047936439514, -1.3241373300552368, 2.321674346923828, 0.055070336908102036, 0.6588159203529358, 0.6974970102310181, 0.16016364097595215, 0.2322075068950653, -0.75893634557724, 1.3708041906356812, 0.7522733807563782, 0.50262451171875, 0.009495428763329983, -1.60566246509552, -1.3700135946273804, 1.0426080226898193, 1.7273937463760376, -0.2740294933319092, -0.3670351207256317, 0.4921550452709198, -0.8869519233703613, -0.8205702304840088, 1.9194552898406982, 0.3373977839946747, 1.0183864831924438, 0.29045364260673523, -0.537593424320221, 1.7201800346374512, -0.27265599370002747, -0.056915055960416794, -0.10360325127840042, -1.3082765340805054, -0.743466317653656, 0.0036293372977524996, -0.28562116622924805, 0.33310097455978394, -0.8385294079780579, -0.6986088156700134, -1.4633599519729614, -0.8646374344825745, -0.7212095260620117, -0.6745232343673706, 1.2745400667190552, 1.1390200853347778, 0.7767421007156372, 0.11287061125040054, 0.02295949123799801, 0.5298161506652832, 0.2032020390033722, -0.9711539149284363, 0.9986404180526733, -0.716035008430481, 0.9107227921485901, 0.32552024722099304, 0.37130799889564514, 0.16351290047168732, -1.090551495552063, -0.4136490225791931, -1.0253527164459229, 0.4251902103424072, 0.9478392004966736, -2.2353248596191406, -0.838969886302948, -0.7492038607597351, 0.055374473333358765, 0.7940636873245239, 0.721799910068512, -0.3891313970088959, 1.8000000715255737, -0.19172751903533936, 0.44978365302085876, 1.1944339275360107, 0.5918006896972656, 0.8362700343132019, -0.5760617852210999, -0.5663011074066162, 0.3114084303379059, -1.1962922811508179, 0.7556966543197632, 0.37151530385017395, 0.021620124578475952, -2.0499887466430664, 0.34441423416137695, -0.9660352468490601, -0.8444845676422119, 0.6820617914199829, -2.029575824737549, 0.13344208896160126, 0.33480530977249146, -0.27962371706962585, 0.08819124847650528, 0.251918762922287, 0.10961528867483139, 0.14221884310245514, 0.5961690545082092, -1.935590147972107, -0.1070924624800682, -1.1855723857879639, 0.4716944098472595, 1.6319900751113892, -0.07436274737119675, 1.1325126886367798, 1.8781964778900146, 0.256971150636673, 0.177265927195549, 1.107311725616455, 0.4501790404319763, 2.145312547683716, -1.8170101642608643, -1.1783441305160522, 0.4301665425300598, 0.1261318176984787, 1.9193079471588135, 0.4435964822769165, 0.924187958240509, -0.3955899477005005, -0.15227851271629333, 0.9322423934936523, 0.9926047325134277, -0.15209653973579407, -1.574019193649292, 0.23002727329730988, -1.2462886571884155, 0.5109497904777527, -0.2145923674106598, -0.052953869104385376, 1.454376459121704, -0.14352568984031677, -1.8104791641235352, 0.04090750589966774, 0.7280836701393127, 0.6394498944282532, 0.4045253396034241, -0.45992133021354675, -0.19122940301895142, -0.13797228038311005, -0.37382546067237854, 1.240061640739441, -1.3953516483306885, -1.0013971328735352, 0.979770839214325, -0.6082169413566589, -1.363735318183899, 0.5566012859344482, 1.3718369007110596, -1.3780289888381958, -0.5073663592338562, 0.5160377621650696, 0.5634018182754517, -1.7839313745498657, 0.7915396690368652, 0.14605043828487396, -0.5106052160263062, -0.788568377494812, -0.6046201586723328, -1.1133222579956055, 0.38727518916130066, -1.8228132724761963, 0.5095186233520508, -0.6406095623970032, -0.5357432961463928, -0.3740324378013611, 0.2482128143310547, -1.1661430597305298, -0.4678311347961426, -0.7688093781471252, 1.2821625471115112, 0.20889486372470856, -0.7871504426002502, -0.4127404987812042, 1.0656499862670898, -1.009577989578247, 2.2650644779205322, -1.281458854675293, 0.35549160838127136, 1.7216168642044067, 1.3820992708206177, -0.1605641394853592, 0.7539194822311401, 2.408754825592041, 0.9903070330619812, 0.6367560625076294, 0.3177366256713867, -0.6325602531433105, -1.320261001586914, -1.5470495223999023, 2.044583320617676, -2.517601251602173, -1.2302988767623901, 0.002651857677847147, -1.8607280254364014, -1.4035497903823853, -0.9620527029037476, 1.1879825592041016, 0.8576151132583618, -0.2210211306810379, 1.793765664100647, 1.806827187538147, -0.33551695942878723, 0.7704347372055054, -0.9827138781547546, 1.897585391998291, 0.4069586992263794, -0.3596953749656677, -0.6765456199645996, -0.5983591079711914, -1.038055419921875, 1.117313265800476, -1.0678339004516602, 0.6764521598815918, 0.9593307971954346, -0.6588164567947388, 0.8565413355827332, -1.616817831993103, -0.04811352863907814, 0.678512692451477, -0.8177132606506348, 0.7013764381408691, 0.5583594441413879, -0.6984781622886658, 0.5386003851890564, -1.2349272966384888, 0.26978492736816406, -1.3683818578720093, -1.4181642532348633, -0.052095599472522736, -0.5993499159812927, -0.6717456579208374, -0.7120804786682129, -1.5501855611801147, -0.07777101546525955, 0.19154271483421326, -1.0765612125396729, 0.7159093618392944, 0.8200863003730774, 0.5701697468757629, -1.3950233459472656, -0.5744006633758545, -0.8354628682136536, -0.8962712287902832, -1.9634672403335571, 1.2387784719467163, -0.543282687664032, -1.0984665155410767, -2.0017683506011963, -0.7094261646270752, 1.4981907606124878, 0.4757860600948334, -0.10786062479019165, 0.4269682466983795, -1.6705061197280884, 0.7716857194900513, 1.2652548551559448, -0.6051433682441711, 0.9205684065818787, -1.2860437631607056, -1.4142314195632935, 0.7067199349403381, 0.316196084022522, 1.7421201467514038, 0.3384481966495514, -1.2495207786560059, -0.6635225415229797, -1.8940114974975586, -0.179128035902977, -0.449219286441803, -1.2051595449447632, 0.9523537755012512, 0.41745203733444214, -0.5429044961929321, 0.15564218163490295, 0.5048206448554993, -1.5271236896514893, -0.006677641067653894, -0.4024132192134857, 1.6923871040344238, 0.8696302175521851, -1.0081586837768555, 0.7697405815124512, -0.9380024671554565, -0.36785101890563965, -0.38968706130981445, 1.129204273223877, 0.7599141001701355, 1.1554620265960693, -0.6968084573745728, 0.6428133249282837, -0.4340314567089081, -1.865545630455017, 1.240377426147461, -0.7930078506469727, -0.2979665696620941, -0.43731430172920227, 0.4534366726875305, 0.9400695562362671, -1.6207945346832275, 0.7741587162017822, -0.8192231059074402, 0.4492270052433014, 0.14062273502349854, 0.7266397476196289, -0.2500704824924469, -1.3708252906799316, 0.40156641602516174, 0.11897479742765427, -1.010865569114685, 0.9854317903518677, 1.6416462659835815, -0.29706066846847534, 0.24598370492458344, 0.02161424234509468, 0.19183388352394104, -1.1208299398422241, -0.6053432822227478, -0.6307616829872131, 1.3431154489517212, -1.3420127630233765, 0.8782814145088196, 0.13037753105163574, 1.22575843334198, -1.46025812625885, -2.1748909950256348, 0.10562369227409363, -1.2299567461013794, -1.4181804656982422, -1.844090461730957, -0.40448132157325745, -0.5211437344551086, -0.7505061626434326, -0.10479812324047089, -1.0141382217407227, -1.6461398601531982, 0.2217969298362732, 0.010945039801299572, -1.2541680335998535, 2.005730390548706, 0.024915222078561783, 1.4373900890350342, 0.544299840927124, -2.428004741668701, -0.025871386751532555, -2.3006625175476074, 0.3959125280380249, -0.9157298803329468, -0.9370493292808533, 1.0805631875991821, 0.6231626272201538, -0.6032808423042297, -0.09229281544685364, 0.12363595515489578, -0.9546651840209961, -0.1667926013469696, 0.5417232513427734, 0.5013796091079712, 0.07233574986457825, 2.000828266143799, -1.0023143291473389, -1.0836193561553955, -1.538219928741455, -1.023746371269226, 0.19345323741436005, -0.7475892901420593, 0.23574481904506683, -1.456362247467041, 0.9846783876419067, -0.2321552187204361, 0.5120836496353149, -2.4972832202911377, 0.3888430893421173, -1.6612955331802368, -0.7605029344558716, -0.5179673433303833, 0.5733559131622314, -1.8582510948181152, 0.8070730566978455, -0.5456281304359436, 0.6843917965888977, 0.22911305725574493, 0.010928316041827202, -0.55938321352005, 0.7620610594749451, -0.6613144874572754, 0.9050872325897217, -0.6912710666656494, -2.7170705795288086, -0.7767110466957092, 1.6380290985107422, -1.0676965713500977, 0.009329065680503845, 0.3100857138633728, -0.4741447865962982, -2.7226808071136475, 0.035201653838157654, -0.9255325794219971, -0.7122790217399597, 0.33392685651779175, 1.0913779735565186, 1.7542033195495605, -0.11921229958534241, 0.4317227602005005, 0.6590474843978882, 1.7620375156402588], [-0.831648051738739, 0.1694362610578537, 0.006957349367439747, 1.1013379096984863, 1.0140509605407715, -1.6884936094284058, 0.8187386393547058, 0.3885815441608429, 0.007956575602293015, 1.0436298847198486, 0.5954083204269409, -0.11855616420507431, 2.26934552192688, -1.8152858018875122, 0.19418518245220184, -0.5022403001785278, 0.9957122802734375, 1.1185411214828491, 0.3427393138408661, 0.1048453152179718, -0.6318184733390808, 0.7676652669906616, -1.1997319459915161, 0.3039226233959198, 0.006859506946057081, -1.4094243049621582, -0.009506420232355595, 0.4846199154853821, -2.1961817741394043, 0.9718470573425293, 0.7383678555488586, 2.0393776893615723, 0.6931996941566467, 1.123970866203308, -0.8483695387840271, 0.5313724279403687, -0.2839019000530243, 0.23100155591964722, 0.3436320424079895, 1.8585413694381714, -0.0783592164516449, -1.4951331615447998, 1.948921799659729, 2.1193909645080566, 0.32445624470710754, -1.322266936302185, -0.7294233441352844, -0.7507398724555969, 1.0280046463012695, -1.0192804336547852, 0.006754183676093817, 1.0518450736999512, -0.2638567090034485, -1.026936411857605, 0.1097543016076088, -1.0876796245574951, 1.3020597696304321, 1.3291164636611938, 0.13476529717445374, 1.207234263420105, -1.4002563953399658, -1.3922252655029297, -0.5450801253318787, -2.150825262069702, 0.8175517916679382, -1.7821495532989502, -0.2791541516780853, -0.5979681015014648, -0.6564042568206787, 0.6391057968139648, -1.6199415922164917, 0.6503835320472717, -1.8231292963027954, 0.13260862231254578, 1.6205300092697144, -0.12514838576316833, -0.2885783314704895, -1.3827468156814575, -0.5394175052642822, 0.07228291034698486, -1.0695700645446777, 0.011939107440412045, -1.4493178129196167, -2.051321268081665, -0.6243593692779541, 1.9576647281646729, -0.9472294449806213, -1.1513981819152832, -1.568129539489746, 0.31893569231033325, 0.04255182668566704, 0.6166930198669434, 0.7430257201194763, -1.3325287103652954, 1.8293991088867188, 1.659716010093689, -1.161199927330017, 1.1458674669265747, -0.8205615282058716, 0.13025039434432983, 0.8990603685379028, 1.3147969245910645, -2.231544017791748, 0.5555250644683838, 0.3276442885398865, -0.39219385385513306, 0.2793455719947815, -1.302268385887146, -1.367438793182373, 0.8951628804206848, -1.2880470752716064, 1.1816245317459106, -0.5303683280944824, -0.03040371648967266, -0.8190852403640747, 0.4464162588119507, -0.9039377570152283, 1.2253751754760742, 1.5116924047470093, -0.8647879362106323, 0.3887837827205658, -0.46867984533309937, 0.6746968626976013, -1.6368135213851929, 1.4823681116104126, -0.18689127266407013, 0.5890922546386719, -1.1760752201080322, 0.2842719852924347, 1.4479998350143433, -2.2763826847076416, -1.8344331979751587, 1.1777760982513428, -1.0295488834381104, -0.6073259711265564, 0.7756621837615967, -0.8545498847961426, 0.1744205802679062, 0.4345756769180298, 0.38113030791282654, -0.10951719433069229, -1.0449110269546509, 0.39559897780418396, -1.0499279499053955, -0.7142850160598755, 0.31608766317367554, -0.17556007206439972, 2.4254183769226074, 0.38728082180023193, -0.38046470284461975, -0.9257431030273438, -0.8418877124786377, 1.9662648439407349, 0.6255982518196106, -0.2796408236026764, 0.7021990418434143, -1.783642292022705, 0.46646949648857117, 0.27835938334465027, 1.530649185180664, -0.43030238151550293, 1.8020464181900024, -1.7230441570281982, -0.6289772391319275, 1.2173079252243042, 2.4624688625335693, -0.7513296008110046, -0.7913365960121155, 0.9774634838104248, 0.06728506833314896, -0.811630368232727, 1.0534052848815918, 0.351747989654541, 0.11321691423654556, -1.6109983921051025, -0.11938530206680298, 0.6943074464797974, 0.8633847832679749, -0.5517358183860779, -0.33501100540161133, -1.4620952606201172, 2.3188726902008057, -0.5060493350028992, 1.0212339162826538, 0.3954426646232605, -0.1336357593536377, -1.4869471788406372, 1.8959685564041138, 0.790473222732544, 1.363050937652588, -0.09233502298593521, 1.1809208393096924, 2.0713613033294678, -1.1999232769012451, -1.0500644445419312, -0.41711920499801636, -0.005823420826345682, -0.3792635500431061, 0.6631466150283813, -0.3831646144390106, -1.0379105806350708, -0.5098373293876648, -0.10009054094552994, 1.3468596935272217, 1.4799903631210327, 0.18452279269695282, -1.0300501585006714, 0.3574207127094269, -0.975638210773468, -2.733003854751587, -0.5936076641082764, -0.4518193006515503, 1.0888718366622925, -0.9123328328132629, 1.4506556987762451, 0.3561338186264038, 0.11206933856010437, 1.3316627740859985, -0.37236568331718445, 0.22288912534713745, -0.9646968245506287, -0.5905513763427734, -0.11603746563196182, 0.9742287993431091, 0.5786973834037781, 0.9846006631851196, -0.49825599789619446, 0.9398149847984314, 0.01283360831439495, -0.8089845776557922, 0.4785721004009247, -0.6929994821548462, -0.01816960610449314, -1.0253247022628784, -1.0585498809814453, 1.2592906951904297, -1.5803488492965698, 1.121146321296692, 0.9909021258354187, -0.4676901400089264, -0.03096107579767704, 2.10300350189209, 0.7790458798408508, -0.627258837223053, -0.5593007802963257, 0.5080137252807617, -0.12404866516590118, -0.11045041680335999, 0.3690970242023468, 0.42531469464302063, 0.23539359867572784, -0.09208065271377563, -0.5950751900672913, -0.3968086242675781, 0.2875933051109314, 0.38119083642959595, 0.0597434937953949, 0.34048134088516235, -0.46931493282318115, -2.2968194484710693, 0.7999714612960815, -0.8994255661964417, 0.7424878478050232, -0.9819220900535583, 0.2630453407764435, 0.7514167428016663, -0.017058532685041428, -0.6473592519760132, 0.10519929975271225, 0.704819917678833, -0.25595811009407043, -1.4386566877365112, -0.5509327054023743, 0.10462834686040878, 1.3495700359344482, 0.7428323030471802, -0.10063739120960236, 0.037601158022880554, -1.447087287902832, -0.8165813684463501, -0.4636095464229584, 0.9627528190612793, 0.8085278868675232, -0.023677371442317963, -1.7864247560501099, -0.656046450138092, 0.36457929015159607, 0.7654931545257568, 1.1256794929504395, 0.25900352001190186, 0.6703948974609375, -0.7922737002372742, 0.4679614007472992, -0.02834327332675457, -0.811126708984375, 1.0606369972229004, 0.2334325909614563, 0.9770823121070862, -0.6880629062652588, 1.1017895936965942, 1.0190623998641968, 0.441204309463501, -0.15478076040744781, -0.5419743657112122, -0.1530962735414505, -0.11642944812774658, 0.7764866948127747, -0.06585422158241272, -1.3141348361968994, 0.11971747875213623, -0.2610336244106293, 0.5353924632072449, -0.043662428855895996, 0.5460220575332642, 1.2278257608413696, -0.9755694270133972, 1.156082034111023, 0.7549436688423157, 0.7930372357368469, 0.23664413392543793, 0.498700886964798, 1.0200611352920532, 0.5081264972686768, -0.3029111325740814, -0.8778734803199768, -0.9028892517089844, -0.06727025657892227, -0.2452271580696106, -1.8623344898223877, 1.0645146369934082, 0.979799747467041, -0.35056138038635254, 0.2736527621746063, -1.1605181694030762, 1.6235452890396118, -0.7374861240386963, -1.5156190395355225, -1.5177524089813232, 1.9720016717910767, 1.7188091278076172, 0.6650002598762512, -1.0581960678100586, -1.1827548742294312, 0.04616234824061394, 1.2050347328186035, 1.6301138401031494, -0.9719106554985046, -0.288003534078598, -0.9094371199607849, 0.8443842530250549, 0.4048001766204834, 1.9609135389328003, -0.36801087856292725, -0.5564974546432495, -0.6501116752624512, 0.42652955651283264, 1.1313544511795044, -0.2721141576766968, 1.2708369493484497, 0.9640451073646545, -1.651809811592102, -0.011817938648164272, 0.5773869156837463, -1.5118873119354248, -0.9818702936172485, 1.1498517990112305, 0.8467201590538025, -0.2145024687051773, -0.37921571731567383, 0.40042418241500854, -1.1796956062316895, -0.9615359902381897, -0.31876981258392334, -2.2982826232910156, -0.37945953011512756, 1.8502728939056396, 0.36220261454582214, 0.23326632380485535, 0.15617366135120392, 0.3294861912727356, 0.7227759957313538, 0.030605755746364594, -0.4703914523124695, 0.8245878219604492, -1.5942659378051758, 0.7477511763572693, 2.111419916152954, 0.3160340189933777, -0.11660046130418777, -1.5807009935379028, -1.0108311176300049, -0.26922574639320374, 0.46525001525878906, 0.5556476712226868, 0.5628487467765808, 0.6715378761291504, -1.5972124338150024, -0.32068052887916565, 0.6687403917312622, 0.6508370637893677, -1.8437426090240479, -0.4683392345905304, -1.150336503982544, -1.3964791297912598, 0.5419353246688843, 0.70033198595047, -1.064470887184143, 0.6165246367454529, 0.8064563870429993, -0.14966538548469543, 1.4360616207122803, -1.4289255142211914, 0.2795632481575012, 0.19435003399848938, 0.3713036775588989, 2.2326128482818604, -0.983242392539978, -1.075884461402893, -0.5752225518226624, 1.051285982131958, 1.8038729429244995, -0.7681140303611755, 0.15697792172431946, 2.0468997955322266, -2.171201229095459, 0.07464113086462021, -0.20317178964614868, -1.6893556118011475, -0.7461008429527283, 0.3886605203151703, -0.02180088683962822, -0.5530136823654175, -0.8911799192428589, -1.3115026950836182, 0.6053337454795837, -0.7558557391166687, 0.49488091468811035, -2.5365118980407715, -0.5981248021125793, 0.7991756796836853, 1.2501327991485596, 0.9941149353981018, 0.33632802963256836, 0.2150474339723587, -1.166602611541748, 0.7358894348144531, 2.2090601921081543, -1.0281692743301392, 0.5713578462600708, -0.3876006603240967, -0.7868258953094482, -0.348239928483963, -0.8284193873405457, -1.031754970550537, 1.4132664203643799, 0.9352989196777344, 1.2803702354431152, -0.7109130620956421, 1.346258282661438, -1.0584007501602173, 1.1400845050811768, 1.3092334270477295, 0.3652329444885254, 1.583604097366333, -0.5153742432594299, -0.4505200982093811, 0.428739994764328, 0.0004955907352268696, 0.3056945502758026, -0.1849159449338913, 0.4857214689254761, 0.7170647382736206, -3.284458875656128, -0.32517918944358826, 0.6757814884185791, 0.8799925446510315, 1.018470287322998, 0.06898902356624603, -1.2832956314086914, -0.20447254180908203, 1.8421603441238403, 0.48498982191085815, -0.6404339671134949, 0.306876003742218, -1.7950907945632935, 0.3860469162464142, -2.0071840286254883, -0.19666330516338348, 0.897098958492279, -0.6119740605354309, 0.5393538475036621, -2.681401491165161, 0.258647084236145, -2.3599956035614014, 0.0917372778058052, -1.2406165599822998, 1.6595640182495117, 1.165460228919983, 1.2287395000457764, 0.9508688449859619, -1.9716768264770508, -0.12382828444242477, 0.2710331678390503, -0.47762125730514526, 1.1013281345367432, -0.03588390350341797, 0.5188159942626953, 0.5915169715881348, 0.5316265821456909, -0.7992939949035645, 2.1854817867279053, 1.0816218852996826, 0.05464324355125427, 1.5752695798873901, 0.46150442957878113, -0.027426566928625107, -0.3698574900627136, -1.0113084316253662, -1.8322049379348755, -0.11314762383699417, -0.15802854299545288, 1.6109209060668945, 0.6183558106422424, 0.40886229276657104, -0.5762622952461243, -0.7644045352935791, -0.18546417355537415, 0.9043897390365601, 0.17853385210037231, 0.4747425615787506, 0.5386658906936646, -0.5464981198310852, -0.03180816024541855, -1.2713462114334106, 0.6526315212249756, 0.11867354065179825, 0.3341202735900879, -1.0753973722457886, 0.6394327282905579, -0.38437792658805847, 1.6813372373580933, -0.3377324938774109, 1.6940863132476807, -0.26829326152801514, -0.4084945023059845, -0.035794902592897415, -0.8995552659034729, 0.8052312135696411, -0.8512883186340332, 1.4247651100158691, 0.6075205206871033, 0.746853232383728, -0.2711779773235321, -0.476712703704834, 1.0247502326965332, 1.8239774703979492, -0.22464580833911896, -2.33855938911438, 0.044619832187891006, -0.1854172646999359, -0.032582107931375504, -0.8458611965179443, -0.31808754801750183, 0.8523920178413391, 0.5038536787033081, 1.3469196557998657, -0.3220253586769104, -0.13546383380889893, -0.6867350935935974, -1.757588267326355, 1.160843849182129, -0.02418540231883526, -0.27312174439430237, -1.1102341413497925, -0.2505187690258026, -0.6460400223731995, 0.25492212176322937, -0.6435072422027588, -0.6988806128501892, -0.30128973722457886, -0.07051476836204529, -0.5315312147140503, 0.7249616980552673, -1.9986059665679932, 0.19875481724739075, 0.223451629281044, -0.26997488737106323, -1.9794248342514038, -1.5207645893096924, -0.8700964450836182, -0.2892853915691376, -0.9482463598251343, -2.5824408531188965, 0.7510635852813721, -0.05254863575100899, 0.6355001330375671, -1.4614007472991943, -1.0552932024002075, -0.5168327689170837, 1.430922031402588, -0.12479018419981003, 0.049511782824993134, -0.6858425140380859, -0.4653501808643341, -1.247930884361267, 1.9136793613433838, 1.5375961065292358, -0.525058925151825, -0.8623875975608826, -2.0841994285583496, -0.2841307818889618, 1.007606029510498, 0.19742201268672943, 0.19030191004276276, -1.1421071290969849, -1.0771280527114868, -1.5942810773849487, -0.17845454812049866, 0.17275799810886383, -1.0965309143066406, 0.19995832443237305, -0.16279318928718567, 0.8519154787063599, 0.9501392245292664, 1.2540606260299683, 0.9981938600540161, 0.5189934968948364, 1.69961416721344, 1.3441952466964722, -0.5064857006072998, -0.2583281695842743, -2.7827601432800293, -0.639338493347168, -0.42909079790115356, 0.6970551609992981, -0.7209777235984802, -2.2547659873962402, 1.1535271406173706, 0.5951457023620605, 0.1183200404047966, -1.3563810586929321, 0.45941364765167236, -1.622960090637207, -0.6659412384033203, 0.024034269154071808, -0.3983023762702942, 0.29801908135414124, -2.0381429195404053, -0.48086580634117126, 1.1254502534866333, 1.161441445350647, -1.631450891494751, 0.006150469649583101, 0.5638052821159363, 0.23150262236595154, 0.7125227451324463, -2.608642816543579, -0.4420616328716278, -0.8416176438331604, 0.9570692777633667, -0.9502615332603455, -0.5102270841598511, -1.2982231378555298, -0.9900697469711304, 0.7827579975128174, -0.15109296143054962, 1.0502307415008545, 1.3257039785385132, 1.9829375743865967, 0.34131693840026855, -0.5921612977981567, -0.9576100707054138, -1.3994557857513428, 0.07993979752063751, 0.1834583282470703, -0.40237465500831604, -0.6093796491622925, -1.660873532295227, -0.45596885681152344, -0.1635400354862213, 0.19679415225982666, -0.04960005357861519, -1.0866554975509644, -0.46456652879714966, -0.016596801578998566, 0.1571267545223236, 0.5044710040092468, 0.03144727647304535, 0.4972938895225525, -1.0865064859390259, -0.4091853201389313, -1.0246071815490723, -0.96406090259552, -1.3520174026489258, -0.2978077530860901, 0.8171247839927673, 1.2480984926223755, -1.8330607414245605, 2.4867193698883057, -0.7841284275054932, -1.9749517440795898, 0.9401133060455322, -1.7070690393447876, -0.7996205687522888, -2.4847021102905273, -0.10545781999826431, -0.578608512878418, 0.5298320651054382, -0.6973934173583984, -1.5835607051849365, 1.5858780145645142, -1.326721429824829, 1.6766812801361084, -1.2388572692871094, -0.29587581753730774, 0.16344328224658966, 0.45739471912384033, 0.8668060898780823, 1.2754524946212769, 1.434689998626709, 0.3835756182670593, -1.0053989887237549, -0.28753724694252014, 0.47045859694480896, -0.9602609872817993, 0.6588053107261658, 0.7343870997428894, -1.6657837629318237, 1.1533757448196411, -1.3442696332931519, 0.8977479338645935, 1.6261192560195923, -0.38762709498405457, -1.2328943014144897, -0.40161141753196716, 0.9062923789024353, 0.6566954851150513, -0.5020828247070312, -0.40361225605010986, -1.9824613332748413, -0.7338405847549438, 2.495373249053955, 1.2268242835998535, 2.3532278537750244, -0.9093971252441406, 0.9203450083732605, 0.7223081588745117, -1.1767053604125977, -1.4121768474578857, 0.004601611755788326, -0.6059492230415344, 0.7998008728027344, 0.29538124799728394, 0.7254581451416016, -0.7056890726089478, -0.7351387739181519, -0.74568110704422, -0.9698963761329651, -1.556695580482483, -1.1251721382141113, 0.6500334739685059, 2.2687666416168213, 0.683123767375946, 0.17225958406925201, 1.8198225498199463, 0.7840775847434998, -0.7176586985588074, 2.6990573406219482, -1.1385602951049805, 1.2449156045913696, -0.8667322993278503, 1.5874251127243042, -0.21662425994873047, 0.40567877888679504, 0.16612079739570618, -0.5985750555992126, 0.30284810066223145, -1.3887689113616943, -2.0687036514282227, -0.7616317272186279, 0.719168484210968, 1.308127522468567, 0.8673660159111023, 1.6745127439498901, 0.35288912057876587, 0.7820810675621033, 1.8122928142547607, 0.11946794390678406, 0.6951675415039062, -0.25743719935417175, 1.1226119995117188, -0.7402028441429138, -0.9375749230384827, 0.123773492872715, -0.8292123079299927, -0.8728411197662354, -1.3797115087509155, 0.8609652519226074, 1.0018770694732666, 1.4594630002975464, -1.3185731172561646, -0.6351861357688904, 1.4006471633911133, -0.47328057885169983, -1.531920313835144, -0.19850684702396393, 1.0503900051116943, -0.39082327485084534, 0.6773822903633118, -1.0196409225463867, 0.0638200044631958, -0.20888543128967285, 0.7745059132575989, 0.9945470094680786, 0.22243840992450714, -0.638501763343811, -1.8961597681045532, -1.2074997425079346, 0.7026861310005188, 0.7612462043762207, 1.0400221347808838, -0.18071997165679932, 1.3994364738464355, 0.18298913538455963, -0.9264413118362427, -0.7569156289100647, 1.8025163412094116, 0.5379195809364319, 0.7489198446273804, -0.3527369499206543, -0.356372207403183, 1.4446508884429932, 0.12898564338684082, -1.2447036504745483, -0.714279294013977, -0.9461037516593933, 0.35330715775489807, -0.2457578033208847, -0.8456019759178162, -0.21585986018180847, -0.15942509472370148, -1.3853944540023804, -0.9746766090393066, -2.1280744075775146, 0.9948093891143799, 0.9097217917442322, -0.37299036979675293, 0.20520125329494476, 0.957146167755127, 0.5758368968963623, 2.5442943572998047, 2.2179818153381348, 0.13707448542118073, 0.28252726793289185, -0.8481671214103699, 1.0932923555374146, 0.19280102849006653, -0.4860287606716156, -0.004856566432863474, -0.8843992948532104, 0.16600799560546875, 1.6942553520202637, 0.03680787608027458, -0.10707516223192215, 0.8468098044395447, -0.01337269227951765, -1.259469985961914, -0.9454445242881775, 1.2310781478881836, -0.9101369976997375, 1.4284694194793701, -0.33403676748275757, 0.8854184746742249, -0.9821099638938904, -0.7266983389854431, 0.534260094165802, 0.7134975790977478, -0.7588427662849426, 0.6195409297943115, 0.6180381774902344, -0.8777148127555847, -1.7377629280090332, 0.7034488916397095, 0.9115410447120667, -1.3801745176315308, 0.5680091977119446, -0.3569091260433197, 1.4552021026611328, 0.9466157555580139, 1.7503693103790283, 0.09468778222799301, -1.1613130569458008, -0.7345200181007385, 0.3544921875, 0.8439797759056091, 1.3512154817581177, 0.08576333522796631, 0.14909756183624268, -1.3778046369552612, 1.6320470571517944, 0.23118549585342407, 0.8234440088272095, -1.803130030632019, 1.0549943447113037, 1.6768097877502441, 0.8771423101425171, -1.5483996868133545, 0.5002008080482483, -0.1946735829114914, -1.1266385316848755, -0.1731814444065094, 0.43110647797584534, 1.4662189483642578, 0.7718473076820374, -1.6515141725540161, -0.43965378403663635, -0.3250524699687958, -0.5580856800079346, -1.6855831146240234, -0.6556835770606995, -2.347351551055908, 1.3251609802246094, 1.0856397151947021, -0.3166932761669159, 0.4424663484096527, 2.0469143390655518, 0.3778892159461975, 0.24974173307418823, -1.7441996335983276, -2.500549793243408, -0.594693660736084, 0.022056549787521362, 1.1382192373275757, 0.2537332773208618, 0.09528230130672455, 0.009908610954880714, -0.615466296672821, -0.8140292763710022, -1.8047105073928833, 0.9319881796836853, -0.8513878583908081, 0.45454299449920654, -1.0355751514434814, 0.2752959728240967, 1.4250223636627197, 0.2899932861328125, 0.8639974594116211, -1.6398253440856934, -0.6210325360298157, -1.178109884262085, -2.038466453552246, 0.10568065941333771, 2.033489465713501, 1.3218132257461548, -0.8551199436187744, 2.383113145828247, -2.319232225418091, 2.477334976196289, -0.48551639914512634, -0.09840262681245804, -1.3792189359664917, 1.30575692653656, 1.790935754776001, -0.21534442901611328, 0.20924216508865356, 0.9216720461845398, 0.14425329864025116, -0.11323164403438568, 0.3244415819644928, 1.6885735988616943, -1.8493428230285645, -0.9034428000450134, 0.5346775054931641, -0.31815123558044434, 0.7450074553489685, 0.34149786829948425, 0.40355822443962097, -0.014643870294094086, 0.3102804720401764, 0.913284957408905, 1.4958558082580566, -1.012591004371643, -0.45160576701164246, -0.12395371496677399, -0.6966358423233032, -1.3173351287841797, -0.9749839305877686, -0.5517587065696716, -0.7714519500732422, -1.1363388299942017, 2.018315076828003, -0.7604661583900452, -0.06896483153104782, -0.010375648736953735, 0.3786119818687439, 0.49744316935539246, 1.0726752281188965, 1.6157938241958618, 0.4202803075313568, -0.011499178595840931], [2.3072493076324463, -0.22978202998638153, -1.4429094791412354, -0.2948983907699585, 1.4342808723449707, 0.06008213385939598, 1.5791093111038208, -0.38217851519584656, -0.800744891166687, -1.6696648597717285, -0.4738486111164093, -0.31747063994407654, 0.5184289813041687, -0.11308880150318146, -1.3838560581207275, 0.679990828037262, 0.0880722850561142, -0.5637345314025879, -0.32231780886650085, 0.547514796257019, -1.5371975898742676, 0.727321982383728, -0.19308380782604218, 0.24938704073429108, -0.8766193985939026, 1.503588080406189, 1.0960137844085693, -0.988540530204773, -1.0556036233901978, 2.140454053878784, 1.5100411176681519, 0.6056551337242126, -0.37891533970832825, -1.3601276874542236, -1.2072010040283203, 1.0618088245391846, -0.2754959464073181, -0.7907466888427734, -0.5852039456367493, 1.138834834098816, 0.2505398988723755, -1.4766267538070679, -0.161950021982193, 0.1917741596698761, 1.1337802410125732, -1.2090545892715454, 0.8686115145683289, -0.5899166464805603, 0.11977028101682663, 0.5643442869186401, -0.4333064556121826, -0.8253152966499329, -1.7195932865142822, 0.8296968936920166, 0.6647970080375671, -1.0556443929672241, -0.08202970772981644, -1.4086543321609497, -0.8899011611938477, 1.1173343658447266, 0.47749388217926025, -0.8823668360710144, 0.4865887463092804, 1.880150318145752, 1.6496002674102783, 0.3343241214752197, 0.6183858513832092, 0.03430530056357384, 0.06159398704767227, 2.4536805152893066, 0.026959365233778954, 1.4160596132278442, -1.062239646911621, 0.45925119519233704, -1.0244561433792114, 0.7413032650947571, 2.78151273727417, 0.9203993082046509, 1.5992120504379272, 1.6452224254608154, 0.6025769710540771, 0.5976683497428894, -0.3194137215614319, 1.2055025100708008, 0.11850079894065857, 0.8012005686759949, -0.9098467230796814, -0.6179670691490173, -1.870046854019165, 2.646214008331299, -0.7877269983291626, -2.5917534828186035, 0.4260387122631073, 1.695267677307129, 0.41653144359588623, 0.6012580394744873, -0.17908155918121338, 1.7520670890808105, 1.4263938665390015, 0.3294278085231781, 1.58212149143219, -1.096921682357788, -1.056094765663147, 1.5674687623977661, -1.5090351104736328, 1.0431190729141235, 0.030965734273195267, 0.31361594796180725, 0.14665701985359192, 1.7300233840942383, 0.7176420092582703, -0.8383113145828247, -0.8694906234741211, -1.4523444175720215, 0.19528894126415253, 0.28011223673820496, -1.080918312072754, -1.045153021812439, -0.9909616112709045, -0.11884108930826187, -1.4691979885101318, -0.5244183540344238, 0.489287793636322, 1.4678819179534912, -1.4156932830810547, -0.1482769101858139, -0.6297036409378052, 0.0832703560590744, 0.29339319467544556, -0.7539788484573364, 1.1071741580963135, -1.4012560844421387, -1.1170988082885742, 0.9138510227203369, -1.218280553817749, 0.154056116938591, -0.4943098723888397, -0.44959595799446106, -0.7158993482589722, -1.7299892902374268, -0.061709627509117126, 1.1618205308914185, -0.3417239487171173, 0.019364967942237854, 2.7293314933776855, 0.5898537635803223, -1.708587884902954, -1.4324235916137695, -0.01734878495335579, -0.076043039560318, -1.3189984560012817, 0.09519191831350327, 0.09928129613399506, 0.4358055889606476, -0.3238834738731384, -1.6210310459136963, -0.662776529788971, -0.5058761239051819, 1.8447171449661255, -0.47558701038360596, 0.023790309205651283, -0.9678044319152832, -0.9313607811927795, -0.6007207632064819, -0.7288613319396973, 1.9001431465148926, -0.9693366885185242, 0.06167115271091461, -0.27326592803001404, -0.9970569014549255, -1.6821788549423218, -0.34253811836242676, -1.3840279579162598, -0.6591100692749023, 0.1632077693939209, 0.14372506737709045, 2.0500104427337646, 0.0701553225517273, -0.21198607981204987, -1.080263614654541, -0.629267156124115, 1.5126073360443115, -1.6261972188949585, -0.23883964121341705, 0.42561110854148865, 0.2583233118057251, 0.692283570766449, 0.4069148898124695, 1.0901713371276855, -0.52623051404953, -1.7351269721984863, -0.09677576273679733, 1.0738539695739746, 1.1463783979415894, -0.3455648124217987, 1.9105224609375, 1.7386200428009033, 0.16609227657318115, -0.48079532384872437, 0.6220439076423645, -0.6521722078323364, 0.16389252245426178, 0.1717091202735901, 0.11804967373609543, 0.35352200269699097, 1.3278783559799194, -0.21300888061523438, 0.05794788524508476, -0.13067469000816345, -0.11606194078922272, -0.08038005232810974, 1.0025708675384521, 1.0724328756332397, 0.262024462223053, 0.39440464973449707, -0.17331553995609283, -0.26299014687538147, -0.1260920912027359, 1.2048357725143433, -0.7057826519012451, -1.3136389255523682, -1.804396629333496, 1.1734449863433838, 0.03027881309390068, -0.5789453387260437, 0.41342660784721375, 2.0346312522888184, -0.2753782570362091, -0.6331276893615723, -0.11849721521139145, 0.3339543342590332, -0.5935093760490417, 0.15020716190338135, -0.9540145993232727, -1.107100486755371, 1.9120560884475708, -0.8893743753433228, 1.6169154644012451, 1.4624054431915283, -3.654334545135498, -0.1889166533946991, -0.43396276235580444, 0.8027573227882385, 0.04369085654616356, 0.63036048412323, 0.7565122246742249, -0.7689397931098938, 0.7454649806022644, -1.8837884664535522, 1.017972469329834, 1.4246443510055542, 1.2667503356933594, -1.2847784757614136, 0.5738223791122437, -0.145847886800766, -1.193544864654541, -0.5772522687911987, -0.36908426880836487, 1.1118258237838745, 0.45566877722740173, -0.8798270225524902, -0.6424769163131714, 0.6176947355270386, 1.1502327919006348, -0.6184029579162598, -0.6496730446815491, -0.05551043897867203, -0.09250922501087189, -0.207641139626503, -0.4712055027484894, 0.6298806667327881, 0.2719707787036896, 1.0485671758651733, 0.7826606035232544, -1.5471910238265991, -0.018886573612689972, -0.7092488408088684, 0.04059014841914177, 1.6932812929153442, -0.8201895356178284, 0.3229946196079254, 0.1909627765417099, 1.0680840015411377, 0.5161378383636475, -0.5915219783782959, -0.5864126086235046, -0.6189871430397034, 1.3196051120758057, -1.704413652420044, 1.2537264823913574, 0.02689482644200325, 0.0547124408185482, -0.7363112568855286, 1.0555070638656616, -0.36952754855155945, -0.43836748600006104, -0.7060750126838684, -0.5493060350418091, -0.0018676409963518381, 0.1255558729171753, 0.9594805240631104, 0.10388477146625519, -0.23891352117061615, 0.09249286353588104, 0.6598992347717285, 0.15216179192066193, -0.5372763276100159, 0.07562772929668427, -1.1159262657165527, -0.15499697625637054, 0.3577394485473633, -0.1878383904695511, 0.5573351383209229, 0.1420096904039383, -0.19625596702098846, -1.9139386415481567, -0.8074736595153809, 0.05620888993144035, 0.2573137581348419, -1.2023289203643799, 1.9591625928878784, 0.9738208651542664, 1.1371526718139648, -0.20831437408924103, 0.7746308445930481, -0.13637717068195343, -1.8593028783798218, 0.503289520740509, 0.02560165897011757, 0.3698224723339081, 0.14013507962226868, 0.8073630928993225, -0.7096475958824158, 0.14977441728115082, -0.5986460447311401, -1.3236206769943237, -0.5886672735214233, -1.0249396562576294, 1.0105518102645874, 0.5514040589332581, 0.05146767571568489, 0.5956058502197266, -0.5706434845924377, 0.8143318891525269, 0.8418775796890259, 1.5295844078063965, -0.16429948806762695, -3.1252264976501465, 0.13754743337631226, 0.003757214406505227, -0.49966874718666077, -0.3489391505718231, -0.7260094881057739, 1.9193215370178223, -2.043509006500244, 0.5214387774467468, -0.9602575302124023, 0.5341887474060059, -0.6191895604133606, -0.5069474577903748, -1.414661169052124, 0.5238215923309326, -0.47819095849990845, -0.023024411872029305, 0.662915825843811, -0.30735844373703003, -0.6344882249832153, -1.124504804611206, 2.3032236099243164, -0.13477948307991028, 1.8807902336120605, 1.0913254022598267, 0.757635772228241, -0.42082446813583374, 0.3054860234260559, -1.698777675628662, -1.1337531805038452, -0.8514977693557739, -0.9267565608024597, 0.026340441778302193, 0.8392915725708008, -0.7051826119422913, -0.6142091751098633, -0.04824351891875267, -0.8258432149887085, 1.7252206802368164, -1.4581388235092163, -0.7173022031784058, 0.03428707644343376, -0.7036524415016174, -1.2622555494308472, -0.47028830647468567, 0.6877319812774658, 0.3968704640865326, 0.5835765600204468, -0.28446832299232483, -0.24609677493572235, -0.47749972343444824, 1.5484309196472168, 0.5555745363235474, 0.5881668925285339, 0.5606383681297302, -0.7955772876739502, 1.5853897333145142, -1.319846510887146, 0.632972240447998, 0.47514021396636963, 0.9098061919212341, 1.3181838989257812, -0.951172947883606, -0.2833895981311798, 0.17287345230579376, -1.075697898864746, 0.44228094816207886, 0.21177822351455688, 0.05645501986145973, 1.0793743133544922, 0.20735114812850952, 0.655012845993042, -0.28141218423843384, 1.2317042350769043, -0.6830506920814514, 0.3267629146575928, -1.420720100402832, 1.701197624206543, 1.1255691051483154, -1.4508024454116821, -0.300367146730423, -0.6866655945777893, 0.19373564422130585, -0.7087470889091492, 0.6593629717826843, 0.8974704146385193, 0.4657497704029083, -1.6342127323150635, -0.26375171542167664, -1.230687141418457, -1.5305757522583008, -0.9903956651687622, 0.36747536063194275, -0.3795149326324463, 1.4826900959014893, -0.23414523899555206, -0.23830364644527435, 0.7366500496864319, -1.8231536149978638, 1.162632703781128, 0.3293362855911255, -1.1858702898025513, -0.34690022468566895, -0.03706054016947746, 1.4274479150772095, -2.220726251602173, 1.7746378183364868, -0.4633732736110687, -2.2713165283203125, 0.6383537650108337, 0.5254559516906738, 1.6509054899215698, 0.9371280074119568, 1.224137783050537, 0.6928437948226929, 0.37427595257759094, 0.3424653708934784, 1.9632004499435425, -2.07370662689209, -0.7914064526557922, 0.6841868758201599, -2.1846539974212646, 0.8603017330169678, -0.0959383100271225, -0.2943662405014038, -0.07604021579027176, -1.550616979598999, -0.4872330129146576, -0.16817477345466614, 0.9857761859893799, 0.1917528659105301, 0.3036683201789856, -0.6324155926704407, -0.3200398087501526, 1.2324204444885254, -1.572762131690979, -1.3681890964508057, 0.47984591126441956, 0.3635439872741699, -0.7379550337791443, 0.05422566086053848, -1.114234209060669, -1.1001629829406738, -1.0492901802062988, -0.566957950592041, 1.6432223320007324, -1.761748194694519, 1.1176754236221313, 1.0456136465072632, 1.050173044204712, 0.4176521301269531, 0.7190442681312561, 2.0064897537231445, 0.3819010555744171, 0.5951785445213318, 0.4580513834953308, 0.3305240869522095, 2.165982723236084, 1.1699167490005493, -2.7683873176574707, 0.700519859790802, -0.2346370816230774, 0.45491883158683777, 0.8151008486747742, -0.4805840849876404, -0.6640876531600952, 0.7855082154273987, 1.255042552947998, 0.41860973834991455, 2.3593742847442627, -0.9082339406013489, -0.5460793972015381, -1.1260675191879272, -0.13694557547569275, -0.0913538783788681, 0.044331710785627365, -0.6560739278793335, 0.3678814470767975, -0.3598890006542206, 0.3215152323246002, -0.034826815128326416, -1.0987275838851929, 1.0334571599960327, -1.2476801872253418, 0.220219686627388, 0.4776807129383087, 1.1587989330291748, -0.0076142107136547565, -1.5298148393630981, 1.1393647193908691, -1.3761773109436035, -0.022249722853302956, -0.08831895887851715, 1.0673575401306152, -0.3621360957622528, 0.7709683775901794, 1.2751197814941406, 0.24843545258045197, 2.4403774738311768, -1.0485951900482178, -0.4316455125808716, -1.3818418979644775, 0.39360833168029785, -0.3109665811061859, 1.5539674758911133, -1.455186128616333, -1.1222072839736938, -0.501853883266449, -0.658056378364563, -1.024553894996643, 1.4570019245147705, 0.8089599609375, 0.1334603726863861, 0.43090546131134033, 0.9021539092063904, 1.4782899618148804, -2.098743200302124, -1.7704704999923706, 0.05851586535573006, -0.7200953364372253, 0.7624605894088745, 0.6131711602210999, 0.8255956172943115, 1.7864489555358887, 0.6972428560256958, 0.4515397846698761, -0.06303201615810394, -0.12570694088935852, -2.272993564605713, -1.0692907571792603, -0.34339234232902527, 0.2669438123703003, 0.2614902853965759, -0.4449339509010315, -0.6776629686355591, 0.13569796085357666, -1.8236976861953735, 0.39678657054901123, -0.31996747851371765, 0.767653226852417, -0.1536213457584381, 1.7962456941604614, -1.7127699851989746, 0.394419401884079, 0.42040374875068665, -0.706151008605957, -0.061417851597070694, 0.9566283226013184, -1.4519580602645874, -1.9090656042099, -0.8287789225578308, 0.3917554020881653, -1.5766186714172363, 1.0924043655395508, 0.04053713008761406, 2.4175169467926025, 0.7086307406425476, 1.6313085556030273, 0.34006038308143616, -1.4678133726119995, -1.1427785158157349, 0.3229626715183258, 0.5819849967956543, -0.15150704979896545, -1.4907094240188599, -0.5770155191421509, 0.45157909393310547, 1.012001395225525, 0.6276417970657349, 0.6000660061836243, -1.119439959526062, -2.06620717048645, -1.469773769378662, -1.3736627101898193, -0.747002363204956, -1.7055495977401733, -0.27140292525291443, 1.2772241830825806, -0.4009550213813782, 0.07669690251350403, -0.2978653609752655, 0.10348659753799438, -1.0256867408752441, -0.9391708374023438, 0.8816319108009338, 2.3186728954315186, -0.5688369274139404, -0.4609416723251343, 0.007935665547847748, -1.1863034963607788, -0.10189361125230789, 0.24955613911151886, 1.557469129562378, 0.34632357954978943, -0.4912782609462738, 0.12813158333301544, 0.26491355895996094, 0.4541303515434265, 0.31976717710494995, 0.45725029706954956, -0.14667503535747528, -1.383285641670227, 0.05737588182091713, -2.5258002281188965, 0.08299709856510162, -0.6578260660171509, 0.3987676501274109, 0.8054530620574951, -1.6320900917053223, 0.28722530603408813, -1.2925328016281128, 1.4882093667984009, 0.37616026401519775, 2.195810556411743, -0.785645067691803, 0.2557108998298645, 0.26786962151527405, 0.6347501873970032, -0.5341073274612427, 0.7358734011650085, 1.8667936325073242, -0.3895840346813202, 0.5518529415130615, -0.024921834468841553, -1.5939829349517822, 0.37677904963493347, 0.25804486870765686, 0.8170871734619141, -1.2634118795394897, 0.06096139922738075, 0.16333073377609253, -0.5554659962654114, 1.010794758796692, -2.119234323501587, 0.6304668188095093, 0.324491947889328, -0.30879634618759155, 0.2985565662384033, -0.6108203530311584, 1.0358679294586182, -2.0506954193115234, 0.36862942576408386, 0.5086244940757751, -1.9135781526565552, -0.6242296695709229, 0.04928145557641983, 0.4582476317882538, 0.6622054576873779, -0.5116298794746399, 0.20015083253383636, 1.9557948112487793, 1.0871131420135498, -1.7923991680145264, -0.10432702302932739, -2.1612377166748047, -0.01493560429662466, 1.1233495473861694, 0.3933328688144684, -0.5736078023910522, -1.3773316144943237, 2.2630629539489746, 0.3020225167274475, -1.8432568311691284, 0.3490811884403229, 0.34714847803115845, -0.27680617570877075, -0.3842966556549072, 0.49962878227233887, 0.728483259677887, 0.5929016470909119, 0.907576858997345, -1.8689438104629517, 1.6661534309387207, 0.9687718152999878, 0.9483145475387573, 0.019206145778298378, 1.2270163297653198, 0.3163454830646515, 0.18041189014911652, -1.040027141571045, 0.2602497935295105, -0.06438075751066208, -2.6208064556121826, 0.27964383363723755, -0.6938385963439941, 0.771955132484436, -1.7779790163040161, -0.5015948414802551, -0.24401289224624634, 0.03830970078706741, 0.30082857608795166, -0.8352450728416443, -0.06804893165826797, -0.804915189743042, -1.360314130783081, 0.4489187002182007, 0.4308781027793884, -1.0817162990570068, 1.4576598405838013, -0.4907359182834625, 0.19463233649730682, -0.11455290019512177, -0.3153632879257202, -0.31411269307136536, 0.15224064886569977, 1.632590889930725, 0.3982935845851898, 0.5000444054603577, 0.3993080258369446, -0.5423200130462646, 0.3721711039543152, -1.6688826084136963, -0.6093178987503052, -1.964407205581665, 0.8404337763786316, -0.45916587114334106, 0.43524712324142456, 1.356305480003357, -0.5497546792030334, -1.2928401231765747, -0.4333941638469696, -0.10444848984479904, 0.46962428092956543, 1.7285066843032837, 0.989719808101654, -1.0283639430999756, -0.7228518128395081, -1.8439010381698608, -1.105217695236206, 0.26943376660346985, -0.8449047803878784, -1.696154236793518, 1.0207157135009766, -0.5311293601989746, -1.1426879167556763, 1.0922889709472656, -0.47380346059799194, 2.485004186630249, 0.7017367482185364, 0.2835056483745575, 0.17279835045337677, -0.55019611120224, 0.5586434602737427, -0.9382592439651489, -1.2823584079742432, -0.03219645097851753, 1.1833409070968628, -0.19141678512096405, 0.12532983720302582, -0.07407571375370026, -0.030812906101346016, 1.201258897781372, -1.7711342573165894, -3.365313768386841, 2.9066312313079834, 0.5725749135017395, 0.17304174602031708, -0.12538452446460724, 0.34063225984573364, -0.3328127861022949, 1.1608591079711914, -0.1312956064939499, 0.5079006552696228, 0.2938551902770996, -0.010499007999897003, 0.26430588960647583, -0.6571608185768127, 0.7017011642456055, -1.4841941595077515, 0.006467194762080908, 1.08559250831604, -0.5059903860092163, 0.15867049992084503, 0.2817445993423462, -1.8242501020431519, 0.8331071138381958, 0.706614077091217, -0.14906221628189087, 0.0007990558515302837, -0.9066634774208069, -1.1467080116271973, -1.44857656955719, 0.9283943176269531, -1.2952015399932861, 0.13152290880680084, 0.39764708280563354, 0.9689393639564514, -0.914897620677948, -0.19456292688846588, 0.6872125864028931, -2.061361789703369, -0.09614607691764832, -0.8771637082099915, -1.0315196514129639, -1.069583535194397, 0.9849862456321716, -1.505967140197754, 1.0836395025253296, -0.009853270836174488, -1.282610297203064, 1.2395055294036865, 1.3509888648986816, -0.12389107048511505, -0.6805802583694458, -0.6263591051101685, -0.2415435016155243, 2.0992610454559326, -0.1650664508342743, -1.3264681100845337, -1.1265785694122314, 0.400825172662735, -0.6202031970024109, -0.8549563884735107, 1.5440928936004639, 0.7475293278694153, -1.961897373199463, -0.39328521490097046, 0.848343014717102, 0.1018570140004158, 0.3285543620586395, 1.6879298686981201, -1.147457480430603, 1.8768858909606934, -1.6256263256072998, 0.5734531283378601, -0.9131581783294678, 0.20640411972999573, 2.605194568634033, -2.0804975032806396, 0.5093300342559814, -0.41273489594459534, -2.738490104675293, 0.37491559982299805, 0.36399024724960327, 0.5137808918952942, 1.3291887044906616, -1.400713324546814, 0.9947199821472168, -0.24705271422863007, 2.043790340423584, -0.1949504315853119, -0.464981347322464, 0.645708441734314, -0.6462641954421997, -2.0070855617523193, 0.7802315354347229, -0.1591368168592453, -0.6084201335906982, 1.403092384338379, -0.5522133708000183, -1.3590775728225708, -0.1825074404478073, -1.1685441732406616, -0.011599247343838215, -0.5285700559616089, 1.8665179014205933, -0.4649907350540161, -1.8708586692810059, -2.3470051288604736, 0.8547887802124023, 0.5824218988418579, 2.540212631225586, -0.9838818907737732, -0.23692192137241364, 1.2738211154937744, 0.22858871519565582, 1.4662026166915894, 1.4519376754760742, -0.7326679229736328, -2.1428589820861816, -0.5973864793777466, -0.22156259417533875, 0.5266990661621094, 0.7028701305389404, 0.35028913617134094, 0.3446844816207886, -0.33538851141929626, -0.45914191007614136, 0.47876349091529846, -0.7313007116317749, -0.7554642558097839, -1.3348544836044312, -1.0529179573059082, -1.458817720413208, 0.561844527721405, -0.4632710814476013, 0.22004230320453644, 0.18155519664287567, 1.087504506111145, 0.20183655619621277, -0.22054335474967957, 0.4630659222602844, -0.24727056920528412, 0.11965315788984299, -1.2409451007843018, 0.795453667640686, 1.0960112810134888, 1.8615410327911377, -0.251020222902298, 0.9931698441505432, -0.5645847320556641, -1.1991060972213745, 1.0259119272232056, -1.105695128440857, 0.30962419509887695, -0.2913486659526825, -1.2328256368637085, 0.11600875854492188, 0.18088380992412567, 0.8871568441390991, -2.476884365081787, 0.10175830125808716, 1.3265407085418701, -0.7614359855651855, -0.18657058477401733, -1.1308658123016357, 1.6219682693481445, 1.2253731489181519, -2.948002815246582, 0.783153772354126, -0.6821920871734619, -1.5872050523757935, 1.6062002182006836, -0.5563532114028931, -0.2533571422100067, -0.7012293934822083, 0.20057648420333862, -1.5357282161712646, 0.8761130571365356, 0.2409093976020813, -0.40943509340286255, -0.8694388270378113, -0.8108084201812744, 1.8346867561340332, 0.20094050467014313, -0.30077263712882996, 0.15873609483242035, 0.7461795806884766, -1.1535484790802002, -0.8239145278930664, 0.8078416585922241, -0.6572535634040833, 0.45360448956489563, -0.9909483790397644, 0.09343942999839783, 0.8417022824287415, 0.42923665046691895, 0.6351965069770813, -0.02733239345252514, -0.7386745810508728, 1.1139636039733887, -0.4435640275478363, -0.7053020000457764], [0.749478816986084, -1.1117234230041504, 1.9370086193084717, -0.6624596118927002, 0.9850413203239441, 1.237006425857544, 1.106574296951294, 0.658150315284729, -0.7138460874557495, 2.7628607749938965, 0.47927403450012207, -0.8669674396514893, 0.001956970663741231, -1.1525858640670776, -1.3728755712509155, -0.2357218712568283, -0.2491753101348877, -2.241975784301758, 0.7104656100273132, 1.0664509534835815, 1.4179311990737915, -0.7446052432060242, 1.2812644243240356, -0.21194244921207428, 1.3301280736923218, 2.0092270374298096, 0.2613750100135803, 0.9434076547622681, 1.5227694511413574, -0.7211422324180603, -1.5570217370986938, 0.6244490146636963, 0.12171262502670288, 0.38203999400138855, -0.7901613712310791, 0.18380290269851685, -1.8754173517227173, 2.1185882091522217, 0.5614583492279053, 0.5750530362129211, 0.8013557195663452, 0.20663417875766754, 0.2728949189186096, 0.35746368765830994, -0.16903801262378693, 0.7294357419013977, 0.04220008850097656, 0.15046726167201996, -0.20091919600963593, 0.15105468034744263, 0.3199829161167145, 0.9733994603157043, -0.6627148389816284, 0.6084312796592712, -1.7357001304626465, 0.19117531180381775, -1.2082974910736084, -0.05083707347512245, 1.2393602132797241, 1.6671509742736816, 1.2410380840301514, -0.7333144545555115, 0.9984138011932373, -0.5099005103111267, -1.390941858291626, 0.8948430418968201, -0.870768129825592, 0.18576261401176453, -0.6746326088905334, 0.7238292694091797, -0.7421215176582336, -0.7823790907859802, 1.9707356691360474, 1.2838499546051025, 0.06636808067560196, -1.6385587453842163, -0.66474848985672, -1.180757761001587, 1.965822696685791, 0.8695624470710754, -0.5250752568244934, 0.5136865973472595, 1.5985037088394165, -0.6651182174682617, 1.1174815893173218, 1.4685801267623901, 0.8405603170394897, 1.2474496364593506, 0.7671910524368286, -0.11955884844064713, -0.10195314139127731, -0.5650558471679688, -1.260831356048584, 0.09947764128446579, -1.045420527458191, 1.7747684717178345, -0.455735981464386, -0.0820079818367958, 1.1520668268203735, 1.237196922302246, 0.9327325820922852, 0.004159760661423206, 0.4297010600566864, 1.3587491512298584, -1.0834625959396362, -0.950747013092041, 0.9344233274459839, -0.5697512626647949, 1.4904834032058716, -0.24662338197231293, 1.4066877365112305, 0.675294041633606, 0.32035425305366516, 0.7079713344573975, -0.2935621738433838, -0.9995477795600891, -0.20075303316116333, 0.49430182576179504, -0.19909054040908813, -0.2341533899307251, 0.8452143669128418, -0.2646733224391937, 1.8210761547088623, 2.0524330139160156, 0.900023877620697, -0.008854924701154232, 1.0634434223175049, -1.1572147607803345, 0.3843876123428345, 0.6071394085884094, 0.4468438923358917, -0.3187241852283478, 0.8757373094558716, -0.014248073101043701, 0.5943650603294373, 2.1326076984405518, 1.5427533388137817, 0.28034985065460205, -0.21147014200687408, -0.5276322960853577, 0.06674441695213318, 2.116659641265869, 1.1898603439331055, 1.1096104383468628, 0.07476145774126053, 1.6441878080368042, -0.5806038975715637, 0.10892044752836227, 0.0033366919960826635, 0.019976714625954628, 0.6993026733398438, 2.4407958984375, -1.754564881324768, -0.5396796464920044, 0.7220132946968079, -2.410869836807251, -0.9876823425292969, -0.2341149002313614, -0.4906112849712372, 0.1688334345817566, 1.653764247894287, -0.05647772550582886, 0.5347372889518738, -1.0566924810409546, 1.1709517240524292, -0.28793197870254517, -0.761002242565155, 0.7381833791732788, -1.7094480991363525, 0.0313502699136734, 1.062274694442749, -0.5613639950752258, -0.01397679839283228, 0.4468332529067993, -1.2357819080352783, 0.634246289730072, -1.3660378456115723, -0.8767610788345337, -0.6915218234062195, -0.5696983337402344, 0.5693368911743164, 1.1660337448120117, -0.2580488920211792, 0.6345683932304382, -0.3327224850654602, 0.1545877307653427, -0.11804499477148056, 0.99346923828125, 1.8593209981918335, 1.463602066040039, -0.4055039584636688, 0.7589492201805115, -0.7576766014099121, 0.6959670186042786, 1.4370946884155273, -1.2136605978012085, -0.34727707505226135, -1.5727264881134033, 0.8341544270515442, -0.030353009700775146, 0.4283893406391144, -0.5221997499465942, 0.7202970385551453, -0.6579747200012207, -0.7313382029533386, -1.5730607509613037, -0.22660858929157257, -0.9508872628211975, 0.6434693932533264, 0.30697163939476013, -1.0350395441055298, 0.5251994132995605, -0.4827364981174469, -0.28509485721588135, 1.82557213306427, 1.389489769935608, -0.48344480991363525, 1.0791233777999878, 0.19992144405841827, -0.7134413123130798, 0.3898169696331024, 0.7135125398635864, 0.2040611207485199, 0.6606472730636597, 0.48002514243125916, 0.4831971228122711, 0.0836872085928917, 0.6236407160758972, 0.7076200842857361, 0.9776037931442261, 1.896236777305603, 0.7783964276313782, 1.7450047731399536, 0.027799192816019058, -0.23471790552139282, 0.4606827199459076, 0.007962062954902649, -0.2969057261943817, -1.456018328666687, -0.9185502529144287, -0.7089805603027344, -0.926115870475769, 0.7848734855651855, -2.403203248977661, 1.0547784566879272, 0.10233809053897858, 0.11459972709417343, -0.06773131340742111, 0.2901889681816101, -0.09870106726884842, -0.5698598027229309, 0.08429030328989029, 0.2772497534751892, 0.757556676864624, 1.4169174432754517, 1.1005924940109253, 0.2873161733150482, -0.26907917857170105, -1.1172783374786377, 0.9620614051818848, 0.976071834564209, -0.8942829966545105, -1.0132447481155396, -0.13042804598808289, 0.8155759572982788, 0.3923284709453583, -0.0008394612232223153, -0.6307276487350464, -1.1782647371292114, 0.1387639343738556, 0.2992725372314453, -0.20170412957668304, -0.25055524706840515, -2.0850048065185547, -1.045646071434021, -0.9493669271469116, -0.4716760814189911, 0.004011800512671471, 1.1090974807739258, -1.7512900829315186, -0.4113289415836334, -0.36700132489204407, 0.5480331778526306, 0.34917423129081726, -1.0864499807357788, -0.349147230386734, 0.3970858156681061, -0.43251195549964905, -0.10196974128484726, -0.06928344815969467, -0.5436312556266785, 0.3233284652233124, 1.2857491970062256, -0.6108642816543579, 0.35468801856040955, 1.6604194641113281, -0.06341345608234406, -0.34389591217041016, -0.12956075370311737, 1.339117169380188, 1.5651342868804932, -0.4760383367538452, -0.9967323541641235, -0.6316835880279541, 0.0843220204114914, -0.628685474395752, -0.26697272062301636, 1.046756386756897, -0.0023659891448915005, 0.4054366648197174, 0.4084187150001526, 0.44697368144989014, 0.3198738098144531, 0.18624581396579742, 0.3290150761604309, 0.33608362078666687, -0.5120386481285095, -0.059496596455574036, 0.5331565737724304, -0.3423103988170624, -1.3105069398880005, -0.7382206916809082, -0.27680808305740356, -0.9483623504638672, -1.4801801443099976, 0.8457183241844177, 0.30457019805908203, -0.7894953489303589, 0.10786344856023788, 0.6970884203910828, -0.14392520487308502, -1.0192315578460693, -0.8242146968841553, 0.7857455015182495, 0.31903088092803955, -0.13107505440711975, 0.17908930778503418, -1.4115642309188843, -0.231667622923851, 0.29545310139656067, 1.4786772727966309, 0.9822962284088135, 0.5540544986724854, -0.7110983729362488, -0.7466980218887329, -1.3789371252059937, 0.26154825091362, -0.9817718267440796, -0.9233635663986206, -0.5440648794174194, 0.6338315606117249, -1.0299122333526611, 0.5796487331390381, 0.9157742857933044, -0.3212708830833435, -0.18321417272090912, -0.5154844522476196, 2.163785934448242, 0.03261168301105499, -0.28430959582328796, 0.27814608812332153, 0.6594815850257874, 0.29446664452552795, -0.4728638827800751, 1.2119066715240479, -1.1255543231964111, 0.5620995163917542, -0.2923940122127533, -0.9506609439849854, -1.2623775005340576, 1.5341602563858032, 1.7214372158050537, 1.6054599285125732, -1.4387569427490234, -0.36817196011543274, -0.5912768840789795, -0.5809956192970276, -0.5920873880386353, 1.2092639207839966, -0.1713525354862213, 1.0282613039016724, -0.8068485260009766, 0.354854941368103, 1.5628471374511719, 0.6249343752861023, -0.946155309677124, -0.6486680507659912, 1.1241992712020874, 0.10710630565881729, 0.3606756925582886, 0.4018159508705139, -0.1577194482088089, 0.36335140466690063, -1.5463486909866333, 1.2025353908538818, 0.32175031304359436, 0.9916079640388489, -0.3480929732322693, 0.3176553547382355, -0.5266703367233276, -0.2501697242259979, -1.476051688194275, 0.493070125579834, 0.0622321292757988, -1.7455910444259644, 0.2892756164073944, 0.38285109400749207, -0.7628282904624939, -1.0068907737731934, -0.8850975632667542, -0.44401928782463074, -1.2081319093704224, -1.1486377716064453, -0.6238327026367188, -1.1715788841247559, 1.1832339763641357, 0.14705847203731537, 1.1550929546356201, 0.7464303970336914, -1.5207209587097168, -2.3147666454315186, -1.1855345964431763, -0.7333151698112488, 0.6389243602752686, -1.4937798976898193, 0.5621715188026428, 1.3609832525253296, -0.44623690843582153, 2.7069075107574463, 1.1962172985076904, 0.5511477589607239, 0.3721374571323395, -0.5169884562492371, 0.014742032624781132, 0.6813649535179138, 0.19907577335834503, -1.0127880573272705, 0.08108682185411453, 1.6036876440048218, -1.3203184604644775, 0.46320605278015137, 3.870405912399292, -1.20477294921875, -0.4982534945011139, 0.21678484976291656, 1.269914984703064, -0.09242726862430573, -1.2557305097579956, -0.08303319662809372, -0.24244624376296997, 1.7794759273529053, -0.24382349848747253, -0.422728568315506, 0.10245916247367859, -2.486711025238037, 5.9690437410608865e-06, 0.5590862035751343, 0.19069983065128326, 1.3737486600875854, 0.5658574104309082, -0.2868093252182007, -2.0258593559265137, 1.5698829889297485, 0.5338631868362427, 0.15657994151115417, -1.173546314239502, 1.0984501838684082, -0.22608502209186554, -0.0607568733394146, 1.8255975246429443, 0.10331789404153824, 0.30314844846725464, -0.9500860571861267, 0.4888887405395508, 0.13834570348262787, -0.4106406569480896, 1.3470858335494995, 1.6704076528549194, -1.0952130556106567, -2.003145694732666, -0.18646827340126038, -0.694010853767395, 0.1823398470878601, -0.02246456779539585, -1.0390714406967163, -0.6639001965522766, 0.6858952045440674, -0.05207805335521698, 0.5304419994354248, -1.0937563180923462, -0.45575442910194397, 1.8885730504989624, -0.2016231119632721, -0.7236751317977905, -0.8762997984886169, -0.03379439935088158, 0.023537205532193184, -1.0643410682678223, 1.7927364110946655, -1.0941402912139893, -0.9173477292060852, 0.2659219205379486, -0.4515153765678406, 0.8509543538093567, 2.1239311695098877, 1.4365341663360596, 0.5537005066871643, 1.3775137662887573, -0.1506115347146988, -2.2822883129119873, 2.2510738372802734, -0.8169445991516113, 0.47647544741630554, -0.8759957551956177, -1.0526570081710815, 0.017163893207907677, 2.4718167781829834, 0.02875535935163498, 0.17038151621818542, -1.4724397659301758, -0.35610318183898926, -1.2062855958938599, 0.1459490805864334, -0.2355540245771408, 1.521655559539795, 0.8877913355827332, -0.8063844442367554, -0.7552749514579773, -1.0918915271759033, 0.3335055708885193, -0.14111189544200897, -0.67001873254776, -0.27667590975761414, -0.1888647973537445, 0.33094552159309387, 0.8388763666152954, -0.2706209123134613, -0.7597128748893738, -0.04787440970540047, 0.8615742921829224, -0.2693452537059784, -0.4031304717063904, -0.03937511891126633, 0.40867745876312256, -0.8752546310424805, 0.1091424748301506, 0.2861444056034088, 0.8425743579864502, -1.4702407121658325, -2.1506168842315674, -0.3723105490207672, -0.3060724437236786, -1.7324814796447754, 1.265772819519043, 0.854358434677124, 0.5778688192367554, -0.48784205317497253, -0.4117920696735382, -0.3843728303909302, 0.6844402551651001, -0.8309915065765381, 0.4572714567184448, 0.3276892900466919, 0.9315605163574219, -0.04566749930381775, 0.9438450336456299, 0.30548620223999023, 0.1169438585639, -0.0548657551407814, 0.09607680141925812, 0.5345564484596252, 0.890234649181366, -0.6088855862617493, 0.38072434067726135, -1.3629499673843384, 0.4642902612686157, -0.5255164504051208, -0.31749144196510315, -0.2879544794559479, 0.17479203641414642, 1.0397149324417114, 1.1304196119308472, 0.16476508975028992, -0.6521852016448975, 0.16991522908210754, 1.0502879619598389, -0.1435241997241974, -0.34325915575027466, -0.10156209021806717, -1.9768610000610352, -0.28303107619285583, -0.15365783870220184, 0.4248865842819214, 0.298361212015152, -0.6053568720817566, 0.09203363209962845, -1.1130683422088623, 1.259798288345337, -1.01030433177948, -0.7249224185943604, -0.2667410373687744, 1.201827883720398, -0.47926661372184753, -1.240332007408142, -1.1374810934066772, -1.8335527181625366, -0.1378920078277588, 0.5770726203918457, 0.7470935583114624, -0.23382778465747833, -0.6753941774368286, -1.6921740770339966, -0.3513394296169281, 0.3471517562866211, -0.6253460645675659, -0.2543829083442688, -1.629138469696045, -0.05937865749001503, -0.3711102604866028, 0.37423640489578247, 0.4307647943496704, -1.1772955656051636, 0.6543543934822083, 0.5052213668823242, 1.4784408807754517, 0.4712459444999695, -0.17832770943641663, -1.4354438781738281, 0.8180949091911316, -0.4161248207092285, 0.2937124967575073, 1.3920649290084839, 0.5703022480010986, -0.47494128346443176, -2.31211519241333, 1.4729549884796143, 0.5008083581924438, -0.02139255218207836, -1.6789659261703491, -0.04680849611759186, 0.6826497316360474, 0.7753415703773499, -0.7717992663383484, -0.7394113540649414, 0.04274723306298256, -0.40308234095573425, -1.6611539125442505, -0.9107741713523865, 0.6161928772926331, 2.2982325553894043, -0.971381664276123, -0.6325045824050903, 0.06409595161676407, -0.3778343200683594, -0.6024200916290283, -2.390388250350952, 0.426219642162323, -0.7692163586616516, 2.561875581741333, -2.0991604328155518, 1.1334564685821533, -0.702192485332489, 0.12515266239643097, 0.12845823168754578, -0.33476364612579346, -0.050687406212091446, 0.5637518167495728, 0.21651622653007507, -0.9171918034553528, -0.17217007279396057, 0.4399755001068115, -0.24785351753234863, 2.1239535808563232, -0.28964126110076904, 0.01903735287487507, 0.7095628976821899, -0.7299510836601257, 0.6436019539833069, 0.7258095145225525, -1.0711584091186523, -0.9866315722465515, -1.548243761062622, -0.4349367320537567, 1.3439807891845703, -0.03349423035979271, -0.7765108942985535, 0.4915529787540436, -0.2872221767902374, -0.8719706535339355, -1.3066308498382568, -0.7242279648780823, -0.7166008949279785, 0.6362035274505615, 0.232876718044281, 0.08368736505508423, 2.008683919906616, -1.6079057455062866, 0.4394652843475342, -0.5846236348152161, 2.6966564655303955, 1.67134690284729, 0.2365298718214035, -0.654965877532959, 1.5394513607025146, 0.8643099069595337, 0.0340871624648571, -0.5817399621009827, -0.552066445350647, -1.6899720430374146, -0.23874184489250183, -0.7872176766395569, 0.9429493546485901, -0.5008513927459717, -2.3030803203582764, 1.1114970445632935, 0.14883218705654144, 0.6685011386871338, -0.8035268783569336, -0.8548474907875061, 1.6068178415298462, 1.1092647314071655, 0.21453212201595306, -0.2857627868652344, 0.061357531696558, 0.645149827003479, -1.8516566753387451, 2.431964159011841, -0.295399934053421, -0.1532403826713562, 1.999849557876587, -0.4724399149417877, 1.3103913068771362, -0.10241811722517014, 0.058578696101903915, -0.9425411820411682, -0.9602614045143127, -0.6552204489707947, -1.1258631944656372, -0.4782004654407501, -0.6017186045646667, 0.14766891300678253, 1.2938566207885742, -0.3204207718372345, -2.3482108116149902, -1.0785075426101685, 0.2691779136657715, -0.6648780703544617, -1.2177355289459229, 0.38519102334976196, 0.4952297806739807, 0.19586414098739624, -0.5506905913352966, -0.4772413969039917, -1.3407188653945923, 0.2379865050315857, 0.6098827719688416, -0.8184846043586731, -1.2201648950576782, -1.9694972038269043, 1.6768516302108765, 0.147661954164505, -0.43933647871017456, 1.0256274938583374, 1.1577496528625488, -0.01971604861319065, 0.4817203879356384, -0.6940860152244568, -0.8851453065872192, 2.86942458152771, -0.309166818857193, -0.6507968306541443, 2.781287670135498, 0.4118425250053406, 0.5051686763763428, 0.9027851819992065, 0.5487924218177795, -0.5465127825737, 0.6978732943534851, -1.8897894620895386, -0.7263308167457581, -1.2625941038131714, 1.1276401281356812, 0.46181342005729675, -0.36440733075141907, 0.8625097274780273, -0.8243597149848938, -0.017276812344789505, -1.4261831045150757, 0.14176851511001587, -1.547491431236267, -1.0724865198135376, -0.6705822348594666, 0.9865386486053467, 1.1064772605895996, 1.317382574081421, -0.07593011111021042, 0.40454086661338806, -0.8865888118743896, 1.0467153787612915, -0.10895334184169769, 0.17817765474319458, 0.5815300941467285, -0.3555227816104889, 2.37725830078125, 1.2161519527435303, 0.8368165493011475, -0.6218254566192627, -0.07044338434934616, -1.4620816707611084, -0.0882827490568161, 1.0393823385238647, -1.2680912017822266, -0.03902667015790939, -2.0567145347595215, -1.819357991218567, -0.43094319105148315, -1.657853126525879, -0.5018191933631897, -0.4630128741264343, -0.012615286745131016, 0.2526683211326599, -0.5391683578491211, 1.1013076305389404, 0.059245068579912186, 1.1298850774765015, 0.3548392653465271, -0.015127202495932579, -0.1151316836476326, 0.10947999358177185, -0.7981447577476501, 0.1567077785730362, 0.23736347258090973, 0.6787797212600708, 1.2077401876449585, 2.040555000305176, -0.4432376027107239, 0.7194688320159912, -0.21171791851520538, 0.4477440118789673, -0.24960152804851532, 0.7947109937667847, 0.4381580352783203, 0.870208203792572, 0.5248897075653076, -1.2984979152679443, 0.6990167498588562, -1.025480031967163, -0.23119403421878815, -2.033083200454712, 1.2804630994796753, -1.5251675844192505, -0.4310556948184967, -0.11460869014263153, 0.3103921711444855, 0.9997122883796692, 1.7067914009094238, -0.1717693954706192, 0.018028870224952698, 1.459058403968811, 0.3913995027542114, -0.45415326952934265, 1.0200175046920776, 0.8541727066040039, -0.28446224331855774, 0.18655073642730713, -0.2854902446269989, 0.4713113605976105, 0.4781055450439453, -0.27911755442619324, -0.1814665049314499, 1.1977179050445557, 0.32449662685394287, 0.32623767852783203, 0.2922140657901764, 1.5014311075210571, -0.25711309909820557, 0.10321509838104248, -0.8277804851531982, -0.6233527660369873, 1.517699956893921, 0.9035778045654297, -0.04053059220314026, 0.905082643032074, 0.029436232522130013, 0.3471825420856476, 0.36840304732322693, 0.013978871516883373, -0.3524388372898102, -0.9958552122116089, -0.4204815626144409, -0.9019445776939392, -0.7174028158187866, 0.6028656959533691, 0.6872023344039917, -0.04167195037007332, 0.17158107459545135, -0.2527415454387665, -0.4450693428516388, -0.13203652203083038, 0.1461954116821289, 0.25639307498931885, 1.4599539041519165, 0.09530110657215118, 1.295276165008545, -0.05028287693858147, 0.3454333245754242, 0.34904348850250244, 1.2173004150390625, -0.8929253816604614, 1.4073206186294556, 0.11584486812353134, 0.0820331871509552, 1.14109468460083, -0.720770001411438, -0.10678564012050629, 0.2908538579940796, 1.0279958248138428, 0.3267623484134674, -0.8698082566261292, -1.0520679950714111, -0.015361189842224121, 0.1684114784002304, -2.1470999717712402, -0.06876381486654282, 0.6503675580024719, -0.1711365282535553, 0.7449855208396912, 1.213368535041809, -1.1679933071136475, -0.18796129524707794, -2.197384834289551, 1.8077619075775146, -1.1424764394760132, -1.743833303451538, -1.5928726196289062, 0.08869562298059464, -0.9176753759384155, -1.6999077796936035, -0.02168644778430462, -0.8518138527870178, 0.6649923324584961, -0.7091413736343384, -0.18608888983726501, -1.688607096672058, -0.6243956685066223, -0.23141713440418243, -0.9749464988708496, 0.381845623254776, 0.0967179387807846, -0.031576886773109436, -0.697048544883728, -0.8218953013420105, 0.22149883210659027, 1.723185658454895, -0.9592244625091553, -0.10856591910123825, 1.2492947578430176, 0.3095908463001251, -0.8520262241363525, -1.6986677646636963, 1.335121750831604, -0.17679287493228912, -1.009504795074463, -0.7778589725494385, 0.17256686091423035, 0.3774632215499878, 0.20623144507408142, 0.41876697540283203, 0.7973225116729736, 0.019005566835403442, 1.7197884321212769, 0.7597585320472717, 0.29659876227378845, -1.2154276371002197, 0.9696151614189148, -0.06975213438272476, 0.7513947486877441, 0.61479252576828, -0.7387226223945618, 1.5515656471252441, -0.9288998246192932, -0.166176900267601, -0.2191774547100067, 0.7729468941688538, 0.19280706346035004, 0.5955904722213745, -0.5284374356269836, 0.18126708269119263, 0.13632039725780487, -0.2669333815574646, -0.17258219420909882, 0.5155118703842163, -1.1070917844772339, 1.3896740674972534, -0.34327784180641174, -1.1744389533996582, 1.7688703536987305, 1.72054123878479, 1.848118782043457, -0.1345832496881485], [-0.39702296257019043, 1.028037190437317, 0.8352905511856079, -1.1427476406097412, -0.29332852363586426, -0.5951890349388123, 0.350485622882843, 1.059340238571167, -0.3882712125778198, 0.31463196873664856, 0.6325123906135559, -0.1402343362569809, -0.7080163359642029, -0.0011417034547775984, 1.0793801546096802, -0.7123672366142273, 0.6878343224525452, -0.9396439790725708, 0.5209329128265381, 0.5662364363670349, -0.730952799320221, 0.4650927484035492, 0.42195308208465576, -1.140578031539917, -1.0124691724777222, -1.60924232006073, 0.5052259564399719, -1.2233831882476807, -0.8301295042037964, 1.2154690027236938, -0.45997482538223267, -1.123019814491272, -0.2082214057445526, 1.3212236166000366, 1.713086485862732, -1.1543833017349243, 0.20945721864700317, -1.7182693481445312, 2.238046884536743, -0.895232081413269, -1.092700481414795, -1.4026153087615967, -1.9372518062591553, 1.0612328052520752, -0.4860459864139557, -1.1184266805648804, 1.5789345502853394, 0.9058794975280762, 1.4471231698989868, -0.6903323531150818, -1.6442725658416748, -1.6493743658065796, 0.3397224247455597, 0.19148606061935425, -1.6937135457992554, -1.1162470579147339, 1.5014818906784058, -1.23134446144104, 0.4272690415382385, -0.8425061702728271, -0.27572375535964966, -1.7243990898132324, 1.7873198986053467, 1.2894295454025269, 1.2202295064926147, -0.2935009300708771, -1.150292158126831, -0.6031349301338196, 1.579471230506897, 0.27175697684288025, -0.48348161578178406, 2.162606954574585, -0.8791545033454895, -0.43696868419647217, 0.5643594861030579, -1.0565307140350342, 0.06815771013498306, 1.4900577068328857, 0.6282668113708496, -1.742181658744812, 1.0955413579940796, -0.2758026421070099, -0.08886592835187912, 0.0509820356965065, 2.0009727478027344, -0.7142388820648193, 1.2970000505447388, -0.8179230690002441, -0.7984793782234192, -1.5210057497024536, 0.30898380279541016, 0.29942432045936584, 0.18643690645694733, -1.800873875617981, -0.21934784948825836, 0.3240046501159668, 0.10531829297542572, 2.376688003540039, 1.2785924673080444, -2.655843734741211, -1.5613682270050049, 0.052222006022930145, 0.2972569167613983, 0.6570395231246948, -0.21455790102481842, -0.04835902526974678, -1.054966926574707, -0.48112666606903076, -1.470342993736267, 0.7631430625915527, -0.3600097596645355, -1.3827602863311768, -0.17127399146556854, -0.027088478207588196, -0.4667913615703583, -0.525249183177948, 1.4325590133666992, 0.2112220823764801, 1.0852235555648804, -1.0358350276947021, -2.009706974029541, 0.3518963158130646, -0.5605688691139221, -0.7808293104171753, -0.8306739926338196, -0.3382963240146637, -0.5117329955101013, -0.19359274208545685, -0.5920574069023132, -0.6164735555648804, -0.8543530106544495, -2.0108139514923096, 1.16713285446167, 0.5211403369903564, -0.339397668838501, -0.205704465508461, -0.2212473303079605, 0.3158700466156006, -0.03332868218421936, 1.49581778049469, 1.8621503114700317, -0.43533438444137573, 0.005406119395047426, 0.8351921439170837, 1.0559451580047607, -1.4763787984848022, 0.6739111542701721, -0.5319231748580933, -2.43192720413208, -1.1521189212799072, -1.0728410482406616, 0.1841963678598404, -0.8359073400497437, 1.628190279006958, 1.2377275228500366, -2.262761354446411, 1.342943787574768, -0.8103327751159668, -1.193191647529602, 1.4648385047912598, 0.8262515664100647, 0.4365202784538269, -0.246406689286232, 0.14283370971679688, -1.156184196472168, 2.1081340312957764, 1.0431772470474243, 0.9130862951278687, 1.4038091897964478, -0.3569074869155884, -0.19058437645435333, 0.03155287355184555, 0.7675982713699341, 0.9532870054244995, -0.8018227219581604, -0.1461823433637619, 1.0311226844787598, 1.2171140909194946, 0.20023903250694275, -0.8163345456123352, -1.0630836486816406, -0.20812730491161346, 1.187369704246521, 0.6332225203514099, -0.5478215217590332, -2.29367733001709, -0.004969841800630093, 0.5653712749481201, 0.29540297389030457, 1.0335222482681274, -0.23576930165290833, -1.0566074848175049, 1.2811880111694336, -0.08715003728866577, -1.5320069789886475, 0.0305654164403677, 0.19338686764240265, 0.24455736577510834, 0.5903634428977966, -0.3292432129383087, -0.19684633612632751, 1.010338544845581, 0.03647014498710632, -0.8492467999458313, -1.626407265663147, 0.038007091730833054, -0.18616844713687897, 0.16532385349273682, 0.02824455499649048, 0.07680830359458923, -0.515903651714325, 0.5303344130516052, 1.1166471242904663, -0.20358514785766602, -1.7798320055007935, -1.0071700811386108, -1.2636102437973022, 0.795203685760498, 0.2331790328025818, 0.12406880408525467, -1.197574496269226, -0.873348593711853, -0.5505883097648621, 0.17256726324558258, 0.4580818712711334, -0.23269833624362946, 1.0528441667556763, -0.7050731778144836, -0.8426900506019592, -0.894250750541687, 0.13723962008953094, -0.015459218062460423, -2.265214204788208, 1.0620183944702148, 0.2541928291320801, 0.47679439187049866, -0.3140919506549835, 0.2329506278038025, -0.13292814791202545, -0.2787180542945862, 0.6418682336807251, -0.6937391757965088, 0.03153371438384056, -1.9808214902877808, 1.1330946683883667, 0.3440456688404083, 1.3717992305755615, 0.30027154088020325, 1.0412957668304443, 1.2189568281173706, 0.07374751567840576, -0.7211993336677551, 2.0797009468078613, -0.7674718499183655, 0.40043312311172485, -0.7897769808769226, -0.37709176540374756, 1.399548053741455, 0.9594228267669678, -0.12923647463321686, 0.41229864954948425, -1.1018978357315063, 0.7036407589912415, 1.1609764099121094, 2.4913153648376465, 0.9289519786834717, -3.142446517944336, -1.174099087715149, 0.215431809425354, 1.8812756538391113, -0.1297011822462082, -2.257324695587158, -0.6509461998939514, 0.6683498620986938, 0.748893141746521, -1.513954758644104, 0.5067983269691467, -0.5179545283317566, 0.20056022703647614, 0.9327617883682251, -0.8384921550750732, 0.34487026929855347, -0.7924221754074097, 0.11689654737710953, 1.4263761043548584, -0.0021641631610691547, -0.3518741726875305, -0.15757280588150024, 0.6350029706954956, 1.5134145021438599, 2.113875150680542, 0.08837643265724182, -0.9712628126144409, -0.014153169468045235, 1.019189476966858, 2.314485788345337, -2.9448482990264893, 0.045779719948768616, 0.7095686793327332, 0.023625588044524193, -0.105087049305439, 0.8448766469955444, 0.04530424624681473, -0.2455160766839981, -0.7361897826194763, -0.3247828483581543, 0.09409549087285995, 1.3260592222213745, -0.08034860342741013, -0.36301836371421814, 0.7146816253662109, -1.0858590602874756, 0.7373117804527283, -0.1553204506635666, 0.45488277077674866, -0.09750929474830627, -0.08358236402273178, 0.33548328280448914, 0.12273474037647247, -1.2487835884094238, -0.5890515446662903, 0.21539613604545593, -1.4750181436538696, 0.4665755331516266, -0.09924469143152237, 1.0688912868499756, 1.0088236331939697, -0.3473338484764099, -1.064194679260254, 1.186141848564148, 0.1531716138124466, 0.7869256734848022, -0.8139356374740601, -0.2127772718667984, -0.0018204232910647988, 0.592352032661438, -0.9511141777038574, -0.07405100762844086, 1.1085959672927856, -1.69887375831604, 0.4956910312175751, -0.6065864562988281, -0.6031466126441956, 0.46986427903175354, -0.10596451163291931, -0.49169206619262695, 0.17353370785713196, 0.18547911942005157, -0.2525077164173126, -0.6006685495376587, 0.1207730770111084, -0.4648703932762146, -0.515649676322937, 0.33723533153533936, -0.6803679466247559, 0.39719292521476746, 1.7036123275756836, -0.8283073306083679, 1.1808561086654663, -1.9841638803482056, -0.3197345435619354, 0.005779807455837727, -0.7866586446762085, -0.6672114729881287, -0.4606526792049408, -0.5401111841201782, -1.9602165222167969, -0.3312872648239136, -1.3533656597137451, 0.4625494182109833, 1.4185799360275269, -1.8716013431549072, 1.2532113790512085, -2.6725120544433594, 0.13258302211761475, 0.25415992736816406, 0.6019459962844849, -0.9032628536224365, -0.3224353492259979, 0.6748717427253723, 0.7174187302589417, 0.18699350953102112, -1.2047003507614136, -0.15827317535877228, 0.024025006219744682, -0.3762587606906891, -0.20986975729465485, -0.28324586153030396, 2.4259960651397705, 0.7517513036727905, 0.05989212915301323, -0.5443370342254639, -1.5685948133468628, 0.7460247278213501, 0.06252943724393845, -0.43457159399986267, -0.6235505938529968, -0.1340562403202057, -0.6172282695770264, 0.40293270349502563, 0.8742372989654541, 0.0457548163831234, 0.2706126272678375, -0.9283052086830139, 0.510671079158783, 2.141568183898926, -1.7678635120391846, 0.7105457186698914, -0.2323724776506424, -0.8198704719543457, 0.0022183305118232965, -0.24359650909900665, 1.231634259223938, -0.0970495268702507, -0.35361990332603455, 1.1553115844726562, 0.7476928234100342, 0.350987046957016, 0.24784207344055176, 1.3252224922180176, -0.38416287302970886, -0.05104135349392891, -0.8822731375694275, 0.9365904927253723, -0.3069625496864319, -0.2065611481666565, 0.36412543058395386, -0.4247918426990509, -2.6281137466430664, 1.0298229455947876, 1.619065523147583, -0.3887825906276703, 0.290250301361084, -0.46440088748931885, 0.16429780423641205, 0.6708344221115112, -0.9831477403640747, 0.15798619389533997, -0.25106415152549744, 0.5122841000556946, 0.6686651110649109, 0.4496587812900543, 0.5029097199440002, -1.9411619901657104, 0.4558161795139313, -2.0103533267974854, -0.35807526111602783, -0.6210422515869141, -2.5817365646362305, -0.44816356897354126, 0.8033974170684814, -0.9049484729766846, 1.5864967107772827, -1.5692696571350098, 1.2645703554153442, -0.41103559732437134, -0.10446583479642868, 1.0189049243927002, -0.41669705510139465, 0.06982956826686859, -0.14945033192634583, -1.1317944526672363, 0.47484052181243896, 0.008270379155874252, 1.3536851406097412, -0.6771520376205444, -0.4058530032634735, 0.3392905592918396, 1.8736239671707153, 0.18446038663387299, 1.510979175567627, -0.181865394115448, 1.0932987928390503, 0.9613538384437561, 1.621229887008667, -1.8299264907836914, 0.4390215575695038, -0.7714962363243103, 0.46940767765045166, 0.056403983384370804, -0.2882467210292816, -0.2409178614616394, 0.13442911207675934, -1.391026496887207, 0.33160415291786194, -0.7932620048522949, -0.7796826958656311, 0.34952038526535034, -0.42466625571250916, 1.9196847677230835, 1.62521231174469, 0.6534788608551025, -0.4749578535556793, 0.23974019289016724, -0.27131107449531555, 1.8445974588394165, -0.35494059324264526, 0.5384731888771057, -1.275043249130249, 0.20200742781162262, 0.4604227840900421, 0.4818231463432312, 0.19790776073932648, -0.3172798454761505, 1.3430514335632324, 0.21227706968784332, -0.32630541920661926, 0.9407180547714233, -0.36973249912261963, -0.47145920991897583, 0.8099803924560547, 2.144897699356079, -0.37876567244529724, -1.2111400365829468, -0.46123334765434265, 0.15463733673095703, -0.132207989692688, -0.41305869817733765, -1.1155256032943726, -0.22469083964824677, -0.4883621037006378, -0.002282700501382351, 0.012652464210987091, -1.2435003519058228, -0.30815958976745605, 0.00580196687951684, -1.6087322235107422, 0.8035106658935547, -0.4942459762096405, -0.6708781719207764, -0.35698944330215454, -0.04980028048157692, 1.1163543462753296, 1.4087920188903809, 0.6983590722084045, 0.3588976263999939, 2.546539068222046, 0.12217948585748672, 1.1053217649459839, 2.0919086933135986, -1.1786471605300903, -0.7382145524024963, -1.294899582862854, -0.21331679821014404, 1.4789730310440063, -0.1212603822350502, -0.8219693303108215, -0.4511314034461975, -0.8135418891906738, -1.0796514749526978, 0.2720441520214081, 1.129926323890686, -0.9000542759895325, 0.2790018916130066, -0.7487542033195496, 0.43901026248931885, 0.6444357633590698, 0.5500593781471252, -0.3588794469833374, 0.4612025022506714, 0.12456898391246796, 1.2491754293441772, 0.0880340114235878, 0.9182196855545044, 0.12133069336414337, 1.397518515586853, -1.0605450868606567, -0.5121601819992065, -0.7256587147712708, 0.26477184891700745, 0.3264026641845703, 0.5064160227775574, -1.0375714302062988, 1.952884316444397, -2.3895349502563477, 0.9118102788925171, 0.8088541030883789, -1.0475720167160034, -0.3617286682128906, 0.5421380400657654, 0.12167680263519287, -0.3940192759037018, 1.116212248802185, 1.1114445924758911, -0.5339808464050293, -0.35225263237953186, -1.7761412858963013, -0.5376853346824646, 1.18040931224823, 0.5607871413230896, -0.988955557346344, -2.1235272884368896, 0.03609558567404747, -0.04691481962800026, -0.7536934018135071, -0.5965023040771484, 0.24336454272270203, 1.2202881574630737, -0.9403705596923828, 0.3610267639160156, 0.06486041843891144, -0.933431088924408, -0.3108633756637573, -0.18237584829330444, 0.659094512462616, -0.06758174300193787, -0.784675657749176, 0.1512213945388794, -0.6323046088218689, -0.9769165515899658, -0.4781627357006073, 0.24749404191970825, 0.853443443775177, 0.7945619225502014, -1.5880869626998901, 0.8422228693962097, 0.5927249193191528, 0.6802836656570435, -2.6819944381713867, 0.2764381170272827, 0.22241471707820892, 1.2934958934783936, 1.2384663820266724, 0.9393656253814697, -0.9939084053039551, 0.36966073513031006, 0.7728206515312195, 1.8980399370193481, 1.599255084991455, -0.03391386941075325, 0.8033441305160522, -0.3598046898841858, 0.7147300243377686, 0.3407779633998871, 0.7026494145393372, 1.4831866025924683, 0.12587377429008484, 2.2669782638549805, 0.28043949604034424, 0.691950261592865, -2.2791635990142822, 0.13435323536396027, -0.40482842922210693, -0.08910949528217316, -1.5820763111114502, -0.7510983943939209, -0.6466275453567505, -0.47012150287628174, 0.896767258644104, 0.4754062294960022, -0.5132167935371399, 0.8062945008277893, 0.6722843647003174, -0.30773621797561646, 0.027238158509135246, -0.8013927340507507, 0.4680086672306061, -1.3719747066497803, 1.0740605592727661, -0.8720754981040955, 0.06911690533161163, -0.8594086170196533, 0.907784640789032, 0.36910614371299744, 1.5380762815475464, -0.08070003986358643, 1.2662992477416992, 0.17630240321159363, 0.049694862216711044, 0.69822096824646, 0.08751518279314041, -0.0807170644402504, -0.8050137758255005, 0.030631253495812416, 0.4858413338661194, 0.37939712405204773, 1.106433391571045, 0.062470823526382446, -0.2375156581401825, 2.014540910720825, -0.8627719879150391, 0.02500808611512184, 0.8057927489280701, -1.3361085653305054, -1.3190972805023193, 0.5439447164535522, -0.633139967918396, 0.057110436260700226, -0.21942000091075897, -0.9350017309188843, 0.5344023108482361, -1.3034067153930664, -0.013564086519181728, -0.4971645474433899, -1.0990172624588013, 0.6573183536529541, 0.14057326316833496, 0.15285611152648926, -1.6147387027740479, 3.3688607215881348, -0.33631888031959534, -0.27641215920448303, -1.4176453351974487, -0.37608540058135986, 1.1516042947769165, 0.45909473299980164, 1.4011023044586182, 0.8935244679450989, 0.2893924415111542, 1.8407292366027832, 0.648952066898346, -0.3221132755279541, -1.4545350074768066, 0.3314827084541321, -0.19883061945438385, -0.07244736701250076, -0.8029950857162476, 0.011342370882630348, 0.3072107136249542, 0.3191632330417633, -0.7394288778305054, 0.0013199198292568326, -0.8608534932136536, 1.4812721014022827, 1.0065250396728516, -1.733702540397644, -0.539385199546814, 0.8133668899536133, 0.4393976330757141, -1.0697318315505981, 0.5392512083053589, -0.5575554966926575, -0.42453432083129883, -1.4382333755493164, -0.16926805675029755, -1.38547682762146, -1.9632611274719238, -1.5148426294326782, 0.33484700322151184, -0.07706093043088913, -0.11778723448514938, -1.0299687385559082, -1.2627187967300415, -0.9329664707183838, -0.6784767508506775, -0.08748193085193634, 0.9366878271102905, -1.2509130239486694, -1.0068293809890747, -1.0325794219970703, 1.9956823587417603, 0.6949044466018677, -0.6222553849220276, -1.3506443500518799, 1.165627360343933, -1.5360528230667114, 2.8350911140441895, -2.177469491958618, -0.8788658380508423, -0.03671732172369957, -0.1569620966911316, 0.3435729146003723, 1.0883501768112183, -1.1386010646820068, -0.181303009390831, -0.7738745808601379, 0.004272807389497757, 0.17403262853622437, -0.04078790917992592, 0.900735080242157, -1.403592824935913, 1.6086732149124146, 1.8982954025268555, 1.6707735061645508, -0.7648230195045471, 0.1651775985956192, 1.8870726823806763, 0.04971809312701225, -1.7102185487747192, -2.0760622024536133, -1.214787244796753, 0.912612795829773, 1.5679230690002441, 1.253433346748352, 1.9109606742858887, -0.7229638695716858, 0.779552698135376, -1.3684475421905518, 0.9077648520469666, 0.16508592665195465, -0.11667994409799576, -1.0636311769485474, -0.456136018037796, -0.7191912531852722, 0.5825619697570801, -0.22069673240184784, 0.6659705638885498, 1.4870024919509888, 2.1634044647216797, 0.8791139125823975, -0.14604994654655457, 2.1084249019622803, -0.03680219501256943, 0.1283821314573288, -0.034476183354854584, -0.8696637153625488, 0.47413507103919983, 1.0700104236602783, -0.4144277572631836, -0.9692097902297974, 0.5705729722976685, -2.409356117248535, -0.5283063650131226, -0.0718822181224823, 0.42013368010520935, 0.17508657276630402, -0.8129932284355164, 1.643509864807129, -0.02535644918680191, -1.0495651960372925, -0.49611544609069824, 0.37474727630615234, -0.13117475807666779, -1.5825586318969727, -0.7639747262001038, 0.3140711188316345, -0.9607661962509155, -0.14608126878738403, 0.771125853061676, 0.4809774160385132, -0.12767821550369263, 0.5858632326126099, 1.3260449171066284, -0.7473101019859314, -0.15956874191761017, -1.0265239477157593, -0.4361524283885956, 0.12566594779491425, -0.6491872072219849, -0.8643304109573364, 0.18746142089366913, 1.7671558856964111, 1.3672187328338623, -0.6287955641746521, -0.01086853165179491, -2.1038565635681152, 0.7716619968414307, 0.3798363208770752, 2.4303030967712402, -0.8702312707901001, -0.4774262011051178, 0.012387339025735855, 0.2250160127878189, -1.7231820821762085, 0.761377215385437, 0.5673854947090149, 0.05755501985549927, -0.6526382565498352, 1.554000973701477, -0.003918012138456106, 0.05845918506383896, -0.7035043835639954, -1.2733761072158813, -0.3728545904159546, -0.46156004071235657, 0.5939894318580627, -0.43115586042404175, 0.21621525287628174, 0.644717812538147, 1.0869758129119873, -0.7242885828018188, 0.7399866580963135, 0.5617531538009644, 0.08685054630041122, 0.6551619172096252, 0.1502140909433365, -0.7397257089614868, 1.8278939723968506, -1.942323088645935, 0.833093523979187, 1.4469969272613525, -0.3474952280521393, 1.124419927597046, -0.3598032593727112, -0.1429138332605362, 1.0456913709640503, -1.6198437213897705, -1.083904504776001, 0.9925855994224548, 0.8675583600997925, -1.4019110202789307, -0.8546690344810486, -0.02936575748026371, -1.235738754272461, -0.20610518753528595, 1.6081377267837524, 0.00010021633352153003, -1.3102883100509644, 0.014421685598790646, -2.3726885318756104, -0.17170631885528564, -0.5945119261741638, -0.01908191107213497, -0.8805809617042542, 0.1380959451198578, 1.935854196548462, -0.2447325885295868, 0.0724913701415062, 3.0348317623138428, -1.358420491218567, -0.8753867745399475, 0.5677648186683655, -0.9820725917816162, -0.8766088485717773, 1.4604487419128418, 0.8584657311439514, 0.5862916111946106, 0.06839405745267868, -0.3874436914920807, -0.3491268754005432, 1.983627200126648, 0.09016074985265732, 0.20965458452701569, 0.25247153639793396, -0.2012881338596344, 0.251772940158844, -0.29984089732170105, 1.4091905355453491, -0.6206459403038025, -0.8322041630744934, 1.4687079191207886, 0.3779790699481964, -1.1665546894073486, -1.4468876123428345, -0.5545916557312012, -1.2701209783554077, -0.1042533665895462, 1.0924209356307983, 0.6095781326293945, 0.4214732348918915, -0.12967705726623535, 1.1282070875167847, 1.096721887588501, -0.9506237506866455, 1.90459144115448, -0.8929479718208313, 0.874467134475708, -0.21728190779685974, -0.5981661081314087, 1.2554619312286377, -0.6295818090438843, -0.3911243975162506, -2.5045278072357178, 0.7943639755249023, -0.7148756384849548, -0.2757374048233032, 0.15118680894374847, -0.0756528452038765, -0.03376047685742378, 0.9128110408782959, 0.8772501349449158, 1.001647710800171, -1.8014781475067139, -0.27363473176956177, 0.8701488375663757, 1.2984843254089355, -1.2267347574234009, -1.0181374549865723, -0.7995090484619141, 0.9131824374198914, -0.6300493478775024, 0.854108989238739, -1.0640240907669067, 0.8936111330986023, 1.164452075958252, 0.5878745317459106, 0.2856481373310089, -0.36126989126205444, -0.6400194764137268, 0.22227515280246735, 1.2512922286987305, -1.8212062120437622, -0.2850123345851898, -0.7015559673309326, 0.10227572172880173, 2.2383618354797363, 0.43668437004089355, 0.9042666554450989, -0.2961733341217041, 0.3517228364944458, 0.4454186260700226, 0.7668154835700989, 0.23270989954471588, 1.2189167737960815, -0.28459376096725464, -1.1310737133026123, 1.0716065168380737, 0.46912452578544617, 0.9237110018730164, -0.27928638458251953], [-1.5270253419876099, 1.4258793592453003, -1.3720242977142334, -0.9700472354888916, -0.08651220798492432, -0.22653041779994965, -0.22724470496177673, 1.2986257076263428, -0.7229905724525452, 2.046630859375, 3.50754714012146, 0.18396775424480438, -0.9652897715568542, 0.15248054265975952, 0.1827632337808609, -0.7977417707443237, 0.8944817185401917, 0.2508276402950287, -0.8078786730766296, -0.17690983414649963, -0.7333763837814331, 0.6350793838500977, -0.7667399048805237, 0.4650766849517822, -2.267385721206665, 0.7499201893806458, -1.0843050479888916, 0.29742181301116943, -0.31464114785194397, -1.0863405466079712, 0.9306862354278564, 0.36934301257133484, -0.8774603605270386, -0.7649234533309937, 0.8795430660247803, -1.0292282104492188, 0.6203081607818604, 1.141474962234497, -0.3263441026210785, -0.04969743266701698, 1.3189321756362915, 0.864878237247467, -0.5448203682899475, 1.7180463075637817, 3.130524158477783, -1.3875004053115845, -0.11238131672143936, 0.3713441491127014, 1.7811706066131592, -0.376189261674881, -0.18946197628974915, -0.5179789662361145, 1.4397282600402832, 0.10184110701084137, -1.849751591682434, -0.6997126936912537, 0.13876046240329742, 1.6613006591796875, -0.4984957277774811, 0.9224717617034912, -0.8618929982185364, 1.133378267288208, -0.16440829634666443, 1.2397865056991577, 0.8995611071586609, 2.4255154132843018, -0.8368654847145081, -0.176443412899971, 1.1254361867904663, 1.5771114826202393, 0.381393164396286, -0.6970039010047913, 0.017378907650709152, -0.3306562602519989, -0.9524853825569153, 0.7588728070259094, 2.3278205394744873, -1.1940940618515015, 1.7209464311599731, 0.4240579307079315, -1.352251648902893, 0.5744659900665283, -0.46009787917137146, -0.4404343068599701, 1.3079718351364136, -3.19085693359375, 0.5028396844863892, -0.04166191816329956, -0.6689162850379944, -1.312137246131897, -0.26546710729599, 1.0243653059005737, -2.1320736408233643, 0.6278902888298035, 1.2368847131729126, 0.9810196757316589, 0.5459458827972412, 1.0353164672851562, 1.3162671327590942, 1.2477794885635376, -1.9156343936920166, 0.6416242718696594, -1.472494125366211, 1.191532850265503, 0.6262957453727722, -0.1776169091463089, 1.7552536725997925, -1.1369487047195435, 0.29167574644088745, -0.8128662109375, -0.004446420352905989, 0.853273332118988, -1.5229779481887817, 0.8489828109741211, -0.7155492305755615, -0.036424241960048676, 0.8256188631057739, -0.09890338778495789, 0.6856327652931213, -0.8349494338035583, -0.11719784140586853, 1.3082507848739624, 0.7207825183868408, 1.8688105344772339, -1.3080947399139404, 0.5001748204231262, 0.20564189553260803, 0.39973005652427673, -0.5590124130249023, -0.7518155574798584, -0.3968774378299713, -0.03898623585700989, 1.3394864797592163, -1.1183762550354004, 2.5139260292053223, -0.16763930022716522, 0.7840338945388794, 0.09862610697746277, -1.7523763179779053, 0.17157314717769623, -1.7342661619186401, -0.35277700424194336, -0.9847797155380249, 0.8459171056747437, -0.009550088085234165, 0.796086311340332, 0.4789818823337555, 2.8670167922973633, 0.8292577266693115, -0.6056097745895386, 1.8138957023620605, 0.017223147675395012, -1.8655608892440796, -1.1119956970214844, -0.40847739577293396, -0.4825104773044586, -0.9924548268318176, -1.0719319581985474, 0.37112343311309814, -1.97383713722229, -0.285041868686676, 0.463162362575531, 0.14149905741214752, -0.19299814105033875, 0.7891638278961182, 1.6794363260269165, 0.24455566704273224, -2.240265130996704, -2.3568477630615234, 1.2308964729309082, -0.4152188301086426, 0.09724406152963638, 0.22586780786514282, 0.39242827892303467, 1.60316002368927, 0.028617296367883682, -0.2070702612400055, 0.3841556906700134, 1.6406633853912354, 1.2340878248214722, -0.7229782938957214, -1.9003130197525024, -0.2911190688610077, -0.14926433563232422, 1.1857210397720337, -0.7447620034217834, -1.752382516860962, 1.1296331882476807, 0.1795102208852768, -1.1687783002853394, 1.8901009559631348, 0.7663275599479675, 1.1914403438568115, -1.0850154161453247, 1.3221768140792847, 0.3860202729701996, -0.7598376274108887, 0.5910763740539551, -0.2407076358795166, 0.6169119477272034, -0.4252161681652069, 0.07155503332614899, -0.1757723093032837, -1.3321696519851685, 0.5265426635742188, -0.083835668861866, -1.5833321809768677, -0.07332393527030945, -2.135444402694702, -0.03166508674621582, 0.08744506537914276, 0.0006818842957727611, 0.6638262271881104, 0.31049299240112305, -0.037194713950157166, -0.8542172908782959, 0.1661999523639679, -0.9803377389907837, -0.7738566994667053, -0.4881350100040436, 2.7100648880004883, -0.7570801973342896, -1.1202226877212524, -0.37564322352409363, -0.9253199100494385, 0.5487583875656128, -0.9687950611114502, -0.27769070863723755, 0.5112515687942505, -1.0806772708892822, 0.3820967674255371, -0.05813765525817871, -0.1322949230670929, 0.689346432685852, 1.6630641222000122, 0.08413521945476532, 0.31475192308425903, 0.4466995298862457, -0.31765666604042053, -1.541496753692627, -0.29700902104377747, 0.054614074528217316, 0.6985260248184204, 1.039924144744873, 0.6971381306648254, -0.07306655496358871, 0.1974080353975296, 0.22025345265865326, -0.22995272278785706, -1.4056633710861206, -0.24121679365634918, 1.4112086296081543, -0.007551269605755806, 0.18773193657398224, -0.8534470200538635, 1.9397058486938477, 1.3051830530166626, -0.0241862703114748, 1.9085252285003662, 1.7747914791107178, 0.22764019668102264, 2.535921573638916, 0.35300856828689575, 0.8281842470169067, -1.9758919477462769, -1.4079378843307495, -0.32371020317077637, 1.0217145681381226, 0.8496138453483582, -1.5917948484420776, -1.4832854270935059, -0.7973127365112305, 0.9766104817390442, -1.899107813835144, 0.7581436634063721, -1.3011094331741333, -0.9803263545036316, 0.029946299269795418, -1.3968263864517212, -0.5437248349189758, 1.632850170135498, 0.029086759313941002, 0.7829283475875854, 0.43817704916000366, -1.5163016319274902, 1.0547888278961182, 0.4550527334213257, 0.861361026763916, 0.012331102974712849, 0.6496002078056335, -1.4289095401763916, 0.5346763730049133, 1.2270077466964722, 0.47771313786506653, -1.3652859926223755, -0.804260790348053, -0.2498723566532135, 0.31632715463638306, -0.6716439723968506, -0.31461119651794434, -1.1870161294937134, -0.03003891184926033, 0.41168761253356934, -0.15116754174232483, 0.6663179397583008, 1.4090927839279175, -0.7783311605453491, -1.8582302331924438, 0.3162248432636261, 0.7136398553848267, 0.7606456875801086, -0.0969168096780777, -1.778254747390747, 0.5856579542160034, -0.2357817143201828, 0.7133811712265015, -0.27996188402175903, -0.11202249675989151, -0.062149569392204285, -0.6658243536949158, -0.4165601134300232, -0.20347586274147034, -1.1841896772384644, -0.7773958444595337, -0.9200253486633301, 1.5318987369537354, -0.06271268427371979, -1.1725833415985107, 0.0632200539112091, -0.05450217053294182, 0.23186710476875305, -2.0117146968841553, 1.9389458894729614, -1.3430393934249878, 0.6064058542251587, 0.014133777469396591, 1.2852892875671387, 0.8359373211860657, -1.4021830558776855, 0.5416314005851746, 1.6554572582244873, 0.5634120106697083, 2.138014554977417, -0.6250085234642029, -0.7853149771690369, -1.3574308156967163, -2.7606449127197266, 0.1852370649576187, 0.8281280994415283, 0.5236588716506958, -0.39400285482406616, 0.913386881351471, -0.30323582887649536, 0.28486698865890503, 0.5854309797286987, 2.186095714569092, 0.4420097768306732, 0.47297337651252747, -0.35180675983428955, 0.23673474788665771, 1.5870757102966309, -1.3213952779769897, -0.29316529631614685, -0.5600780844688416, -0.1524772047996521, -0.36891114711761475, -0.32723715901374817, -1.6639524698257446, -1.9640260934829712, 0.35887017846107483, 1.8452367782592773, -0.18392927944660187, 0.32334038615226746, -0.11687145382165909, -1.2086256742477417, 0.445069283246994, -0.26238206028938293, -0.9632891416549683, 0.08353716135025024, -0.30476275086402893, -0.8601678609848022, -0.7115743160247803, -0.04697059094905853, 0.2792225778102875, 2.006934404373169, -1.4137591123580933, 0.7881448268890381, 0.07649324089288712, 2.335597515106201, -0.25093966722488403, 0.5313761234283447, 2.0578536987304688, 0.16642023622989655, -0.4554475247859955, 1.7143810987472534, 0.06485015153884888, -0.4451848268508911, 1.1550273895263672, 0.7549257874488831, -2.8598697185516357, -0.45393434166908264, -0.015635527670383453, -0.5480533242225647, -1.5483981370925903, -1.1044296026229858, -0.6542211771011353, 1.2761503458023071, 1.3800276517868042, -0.9304360747337341, 1.1350857019424438, 1.3331100940704346, 0.15999656915664673, -0.8675675988197327, -1.7548837661743164, -2.5123825073242188, 0.6115716695785522, 0.45093658566474915, -1.0585544109344482, 1.2520921230316162, -1.8989936113357544, -0.22709113359451294, -2.1834073066711426, 0.6024608016014099, 0.5339161157608032, 1.4659961462020874, -0.3571798801422119, 0.2518408000469208, -1.0175652503967285, -0.00011165442992933095, -0.38993948698043823, 1.617219090461731, 1.293511986732483, 0.3447559177875519, -0.8280248641967773, 0.36552250385284424, 0.5893420577049255, 0.6932491660118103, -0.382779061794281, 0.06418110430240631, 0.17804881930351257, -0.8265806436538696, 0.8822168707847595, 1.729993462562561, -0.5634770393371582, -0.05811340734362602, -0.2746487259864807, 1.1388241052627563, -1.0160552263259888, -0.09434328228235245, -1.4252278804779053, 0.3366079032421112, 0.6763116121292114, 0.9637736082077026, 0.04151638597249985, 0.4737449586391449, -0.3648182451725006, 1.963637113571167, -0.2634654939174652, -0.313190758228302, -1.7426872253417969, 0.48700135946273804, 1.642155647277832, 0.9344305992126465, 0.2958102226257324, 0.007219544146209955, 0.5510737299919128, 1.143412709236145, -0.4561559855937958, -1.2107239961624146, 0.33600661158561707, 0.5140392780303955, 1.45905339717865, 0.19631865620613098, -0.7282319068908691, 0.10342710465192795, 0.9559435248374939, -0.8086644411087036, -0.6926156878471375, -0.772763729095459, 1.2365305423736572, -0.7599909901618958, -0.009451404213905334, -1.0146459341049194, -0.8975378274917603, 0.1590937227010727, 0.2153690606355667, 0.4315764605998993, 0.06510352343320847, -0.5933701992034912, 0.965718686580658, 1.1620187759399414, -0.12782864272594452, -1.5009326934814453, 1.2137763500213623, -1.1103932857513428, -0.8838528990745544, -0.17788314819335938, -0.8862730264663696, -0.03791067749261856, 0.9196889400482178, -0.6589508652687073, 0.051232483237981796, 0.36309266090393066, -0.684579074382782, 0.07406499981880188, -0.5323559641838074, 0.8769583702087402, 0.03612363338470459, 0.23817746341228485, -0.14916527271270752, 0.6498920321464539, -1.0469138622283936, -0.6061874628067017, 1.1077802181243896, -0.8362034559249878, 0.13362452387809753, -0.7096979022026062, 1.8496417999267578, -0.49390628933906555, -0.06590906530618668, -1.0806901454925537, 1.0586798191070557, 0.1860658973455429, 1.8122235536575317, 0.6720791459083557, -0.34788477420806885, 0.30437344312667847, -0.004530320875346661, -0.6931992173194885, -0.7113840579986572, -0.17052505910396576, 0.37093600630760193, -0.2109820693731308, -0.10212061554193497, 0.0883064717054367, 0.47982436418533325, -0.7464056611061096, 1.2562450170516968, 0.08261837810277939, -1.061501145362854, 0.6040266752243042, -0.30632284283638, 0.5638190507888794, -1.1378602981567383, 0.45761099457740784, -0.06923023611307144, -0.12429165095090866, -0.08779381215572357, -0.32239845395088196, -0.9233136177062988, 1.1484391689300537, -1.5931556224822998, -1.237263798713684, -0.8138220310211182, -0.3121040165424347, -0.2132168412208557, 0.7111421823501587, 0.39017200469970703, 0.8746827244758606, -1.2819335460662842, -1.1565706729888916, -0.8941420912742615, 1.4870485067367554, -0.7825596332550049, -1.1883500814437866, -0.49135541915893555, -0.07514245063066483, -1.5546234846115112, 0.7419178485870361, -0.20508448779582977, -1.094720482826233, -0.23288863897323608, -0.5862424969673157, 1.198104977607727, 0.020494384691119194, 1.0568082332611084, 1.2502199411392212, -0.787230372428894, 0.22392341494560242, -0.04819020256400108, 1.3595049381256104, 1.6301932334899902, -0.7506672143936157, 0.09690805524587631, 2.814575433731079, 0.009510235860943794, -0.46354642510414124, 0.6262606382369995, 0.5535620450973511, 1.4760208129882812, 0.890417754650116, 0.1021440252661705, -0.4169364273548126, 1.5489356517791748, -2.126917600631714, -1.496310830116272, 0.9742264151573181, 0.3060210943222046, 0.767873227596283, -0.17234359681606293, 0.09576359391212463, 0.34197095036506653, -1.0646491050720215, -0.24116356670856476, 0.6020494699478149, 1.6524200439453125, -0.07577241212129593, -1.263839602470398, -0.04156671091914177, 0.13840892910957336, -0.6367340683937073, 0.9358275532722473, -0.5262043476104736, -1.531691312789917, 0.8823389410972595, -0.27222147583961487, -0.1845715194940567, -0.9966833591461182, -1.3447270393371582, 0.5916457176208496, 0.1336068958044052, -0.7375980019569397, 1.900020956993103, -0.9795507788658142, 1.4210357666015625, 3.39583683013916, -2.2205722332000732, -0.5564044117927551, -0.46089789271354675, -0.45291122794151306, -1.8505682945251465, -0.482321172952652, 0.19620223343372345, -1.167248010635376, 1.0273550748825073, -1.042362928390503, 0.9481615424156189, 0.40844085812568665, -0.567091166973114, 1.3945189714431763, 0.18393269181251526, 0.5231418609619141, 0.2898521423339844, 1.9070329666137695, -1.188514232635498, -0.5892805457115173, 0.21681031584739685, -0.42798787355422974, -0.3484264314174652, 0.541312575340271, -2.2012922763824463, -0.9897902607917786, 0.6010698080062866, 1.6759519577026367, 0.22185546159744263, 0.43824535608291626, -0.3741481304168701, -1.1459286212921143, 1.2948898077011108, 0.3328898847103119, -0.08719480037689209, -0.34271085262298584, 0.9468286633491516, 1.2348190546035767, 0.10456424951553345, -0.5080775618553162, -1.8638683557510376, 0.07151734828948975, 0.1964605152606964, 2.026050567626953, -0.7096524834632874, -0.84027498960495, -0.6466071009635925, -0.3449493944644928, 0.2602926790714264, 0.2729673385620117, -0.3359576463699341, 0.5889419317245483, -1.4591776132583618, 1.533945918083191, 1.2089340686798096, 0.3647613227367401, -0.8901666402816772, -1.00780189037323, -0.33231908082962036, -1.2572568655014038, -0.056458890438079834, -1.1079365015029907, -0.035551898181438446, -1.1572597026824951, -0.17201955616474152, -0.1459505707025528, -0.08075971901416779, 1.05091392993927, -1.5085891485214233, -0.2362286001443863, 1.0685254335403442, 0.1581648737192154, 0.044751446694135666, -0.5026317834854126, 0.5527536869049072, -0.11738285422325134, 0.9788142442703247, -0.1714184433221817, -1.2332408428192139, -0.36158818006515503, -1.6690874099731445, 2.5996670722961426, 0.28160232305526733, 0.42442283034324646, 1.7837706804275513, -2.228633165359497, -0.1736503541469574, -0.08491484820842743, -0.4160989820957184, 1.1336227655410767, 1.3922264575958252, -0.6193355321884155, 0.3088110089302063, 1.5837894678115845, 0.0003781006671488285, -0.04852079227566719, 0.7178764939308167, -0.13566800951957703, -0.3255574405193329, -0.5542929768562317, -1.341863751411438, 0.7600593566894531, -0.6969107985496521, -1.3024941682815552, 2.779130458831787, 0.5558731555938721, 0.10713133960962296, 1.154832363128662, -0.7480404376983643, 0.36456596851348877, 0.2956486940383911, -1.4212841987609863, -3.266430616378784, 0.38078930974006653, -1.4100229740142822, -0.4145768880844116, -0.2978580594062805, 0.8429321646690369, -0.905677855014801, -0.3277958035469055, -2.6044704914093018, -0.051875434815883636, 1.4282864332199097, 0.5041796565055847, 0.08984063565731049, 1.2718791961669922, -0.815757155418396, -0.9365329146385193, -1.0589243173599243, 0.3841485381126404, 0.3725281059741974, 0.3933146595954895, -0.9823645949363708, -1.3136489391326904, 0.6453107595443726, 0.6556123495101929, -0.6804744005203247, 0.7106640934944153, 0.3293512165546417, -0.3935435712337494, 1.1160751581192017, 0.3622385859489441, 0.76301109790802, 0.19618985056877136, -1.247331976890564, 0.045587506145238876, -0.29136595129966736, -3.014782428741455, -0.9505956768989563, -1.1708823442459106, 1.0573303699493408, 1.043352484703064, -1.6560425758361816, -0.36777088046073914, 1.0514708757400513, 0.9620435237884521, 0.9601574540138245, 0.666227400302887, -1.696520209312439, 0.800494909286499, 1.1663178205490112, -0.9377752542495728, 0.8734560608863831, -0.1961708664894104, -3.0353429317474365, 0.47375205159187317, 0.16332821547985077, -0.7011796236038208, 0.2867605984210968, 1.3389313220977783, -0.129135400056839, 0.9462199211120605, 1.046858549118042, -0.4851895868778229, -0.8546726703643799, 0.9083194136619568, 0.5060709714889526, 0.6000659465789795, 0.043945133686065674, 0.16667619347572327, -0.40342891216278076, -1.666593074798584, -0.23399800062179565, -0.18682508170604706, -0.9830226898193359, 1.0253783464431763, -0.09313801676034927, 0.4362567663192749, 0.9027559161186218, 0.9185585379600525, 1.7172430753707886, 0.8285260796546936, -0.4313686192035675, -1.4152321815490723, 0.5882902145385742, -0.046937495470047, 0.9813225269317627, -0.6142476201057434, 0.8347190022468567, 0.2503683865070343, 0.6029084324836731, 0.8934450149536133, 0.7179871797561646, -0.7435137629508972, 2.352984666824341, -0.5963243246078491, -0.10348326712846756, -0.5102642178535461, 0.363407701253891, 0.3988744914531708, 1.923423409461975, -0.2608776390552521, -1.8137050867080688, -0.8706943392753601, -0.3592754006385803, -0.9424593448638916, 2.0943920612335205, -0.3146860897541046, -1.2026335000991821, 0.9078708291053772, 0.05818253383040428, -0.9505605697631836, -1.087989091873169, 0.7620225548744202, -0.9736064672470093, 0.7146031260490417, -0.6621243357658386, -0.8199167251586914, -0.9060863256454468, 0.08349400013685226, 1.2720458507537842, 0.43183568120002747, -1.252341628074646, 0.3619893789291382, -0.9651728272438049, -0.6010984182357788, -1.0373626947402954, -2.2541849613189697, -0.31297147274017334, 0.2396879643201828, 0.8998527526855469, -1.1637600660324097, -0.099242202937603, 0.47697770595550537, 0.7124998569488525, -0.7660337686538696, -1.1183706521987915, 1.7073349952697754, 0.34588178992271423, 0.3179640769958496, 0.9328866600990295, -0.9649426341056824, -0.5445596575737, 0.4521101117134094, 0.9675278663635254, -1.6620453596115112, -1.0011398792266846, 1.8661394119262695, 1.8535971641540527, 0.8220295310020447, 1.3947093486785889, 1.2690644264221191, 1.7087514400482178, -0.5627294778823853, 0.7720924019813538, -0.20871707797050476, 0.9396753907203674, 0.17567920684814453, 1.2783825397491455, 0.7975015044212341, 1.2430018186569214, 1.130467414855957, -0.7665237784385681, -2.8170721530914307, -0.5445161461830139, 0.547359049320221, 1.053244709968567, -1.100658893585205, 0.4100615978240967, -0.587201714515686, -0.4641391932964325, 0.3533548414707184, 0.05494050309062004, -2.070197820663452, -1.0019887685775757, 1.0937047004699707, 0.38132137060165405, 0.3303987681865692, -0.6597543954849243, 0.30661067366600037, 0.7486066222190857, 0.7366723418235779, -1.1370527744293213, 0.28713199496269226, 0.25598740577697754, 1.6184556484222412, 2.1166419982910156, 0.6010643839836121, -1.286515474319458, 0.27948257327079773, 0.04440204054117203, 0.4407474994659424, -1.2827973365783691, 1.6770368814468384, 0.1501399278640747, 0.05636530742049217, -0.28540167212486267, -0.5621939301490784, -0.29874804615974426, -0.20682547986507416, 0.951789915561676, 0.27709493041038513, 1.4141641855239868, -0.5367159843444824, -0.785183846950531, -0.5959543585777283, 1.3391518592834473, -1.4829546213150024, 0.8398464322090149, 0.9093202352523804, 0.7899539470672607, -0.8504437804222107, 0.23687544465065002, -0.3794712424278259, 0.42915746569633484, -0.423377126455307, 1.3686699867248535, -0.4182635247707367, 2.0953962802886963, -1.2546700239181519, 0.1382787823677063, -1.5253934860229492, -1.3363354206085205, 0.6567021608352661, -1.388023018836975, 1.8553828001022339, 0.15325693786144257, 0.0056481873616576195, 0.6869458556175232, -0.4109478294849396, 0.4866477847099304, -0.5671637058258057, 0.3013017475605011, 0.40655142068862915, 0.08603723347187042, 2.0716710090637207, 0.17649513483047485, 0.7116835117340088, 0.6263860464096069, -1.7075375318527222, 0.12446234375238419, 1.5106201171875, -0.4097953736782074, 1.6995245218276978, -1.0562161207199097, 0.6954320669174194, -0.8842695951461792, 0.5704014897346497, -1.8505494594573975, 1.8499770164489746, 0.7946015000343323, 0.8739621043205261, -1.728566288948059, -0.12445513904094696, 0.3897711932659149, -0.6930869817733765, -0.9794094562530518, 2.101825714111328, 0.043900277465581894, -0.9959725737571716, 0.7956855893135071, -0.0726483091711998, 0.483447402715683, 0.17984582483768463], [1.1210200786590576, -0.4944019615650177, 1.6941442489624023, 0.8802629113197327, 0.766071081161499, -1.2656512260437012, 0.34810107946395874, 0.5169592499732971, -0.21464183926582336, -0.24461284279823303, 1.021445631980896, -1.2076891660690308, 0.45469236373901367, -1.806484341621399, -0.5260049104690552, 0.030542992055416107, -0.19688260555267334, 0.7900124788284302, 2.0272374153137207, -0.7099835276603699, -0.485886812210083, 0.29497554898262024, -1.2286015748977661, -0.37677061557769775, -0.8382651805877686, -0.33219853043556213, -1.3567373752593994, 0.014523995108902454, 0.39039915800094604, 0.9896479249000549, 0.48922407627105713, -1.4303834438323975, -0.9160318374633789, 1.1372277736663818, -1.334017276763916, 0.4601871073246002, -0.31360924243927, -0.27730944752693176, 1.1291598081588745, 0.5917037725448608, -0.7103298902511597, 1.0317304134368896, 0.4753546118736267, -0.16631031036376953, 1.2313649654388428, -0.37274816632270813, -0.8402473330497742, 1.0707781314849854, -0.07231316715478897, 0.13136127591133118, 0.2929700016975403, 0.8034980297088623, 0.35118648409843445, 0.3171759247779846, 1.304513692855835, 1.6132463216781616, 0.5836790204048157, 1.509366750717163, -0.13522639870643616, 1.0569417476654053, -0.2362254112958908, -1.621878981590271, -0.7235310077667236, -0.7507264018058777, -0.3760102093219757, -1.0096708536148071, -1.672743558883667, -1.9618823528289795, 0.3139989376068115, 0.6286670565605164, -1.187983751296997, 0.12777677178382874, -1.53147292137146, -0.08539470285177231, 0.5614787340164185, 1.5721473693847656, -0.4930596351623535, -0.7387358546257019, -0.4709518849849701, 1.0822919607162476, -0.273980975151062, -0.44920337200164795, -1.4240094423294067, 1.7662699222564697, -0.4712650179862976, 0.4748515188694, 0.37047645449638367, -0.5087930560112, -1.4388867616653442, -0.6760165691375732, 0.32779231667518616, -1.8743622303009033, -0.35112228989601135, 0.7781631350517273, 0.398459792137146, 0.9868377447128296, -0.04877983778715134, -0.023086750879883766, -1.1213911771774292, 0.0522795170545578, 0.31767207384109497, -1.4148844480514526, -0.6568583846092224, -0.040375106036663055, -1.0508687496185303, 1.2216180562973022, -0.05090264976024628, 0.16112209856510162, -0.1113695427775383, 1.0970053672790527, -0.8247791528701782, 0.03609194606542587, -2.1847972869873047, -0.5405998826026917, 0.6334793567657471, 0.31613242626190186, 0.7922373414039612, 0.11171437799930573, -0.20130205154418945, -0.07739194482564926, -0.5164373517036438, -0.034120943397283554, -1.8267076015472412, 2.3299341201782227, -0.8995358347892761, 0.1098465770483017, -0.30706650018692017, -0.8324835896492004, -0.5029832720756531, 1.0302069187164307, -0.2701612412929535, 0.1305629014968872, -1.093937873840332, 1.225455641746521, 0.2938806414604187, 1.0100311040878296, 0.43844324350357056, -0.4228293299674988, -0.7897989153862, 0.5437244176864624, -1.9019591808319092, 1.9121079444885254, 1.8347288370132446, 0.5452291965484619, 1.9845155477523804, -2.186718463897705, 1.000476360321045, -0.05098545178771019, -0.7362506985664368, -1.9097721576690674, -0.21900860965251923, 0.9752036929130554, 0.32109156250953674, 0.993744432926178, -1.0601791143417358, 0.33418238162994385, 0.3407956659793854, 1.8432402610778809, -0.648684561252594, -1.474676489830017, 0.9525296688079834, -1.0732107162475586, 0.28316646814346313, -2.1323344707489014, -0.34745919704437256, 1.164841890335083, -0.6108936667442322, -1.4641298055648804, -0.283700555562973, -0.16863258183002472, 0.805459201335907, 0.36565324664115906, 0.34340232610702515, -1.2132599353790283, 1.0695205926895142, -0.07347878068685532, -0.09571492671966553, 0.7050536870956421, 0.5134024620056152, 0.8524616956710815, -0.45918965339660645, 0.36666908860206604, -1.1259167194366455, -0.9667643904685974, -0.3168831169605255, 0.7524615526199341, 0.7217524647712708, 0.551105797290802, 0.6415954232215881, 0.10167370736598969, -0.39390039443969727, -0.47827816009521484, 1.240782618522644, -0.7003089785575867, 2.246331214904785, -0.5802127122879028, 0.7290546894073486, 0.33946898579597473, 2.498847246170044, -1.085432529449463, 0.17778615653514862, -1.144879937171936, 0.04428098350763321, 0.1240796223282814, -1.7182899713516235, -0.9635859131813049, 0.13823740184307098, 1.1614183187484741, 0.3858298659324646, -0.18339432775974274, -0.07097230851650238, -0.9623476266860962, -1.1020466089248657, 0.8557361960411072, -0.13860423862934113, -0.5764410495758057, -1.1297146081924438, -2.006310224533081, -1.2586110830307007, -0.14249098300933838, 1.6966396570205688, 0.388582319021225, 0.34703758358955383, -0.4785759150981903, 1.1591861248016357, 1.8447247743606567, 0.9833380579948425, -0.0016217799857258797, -0.8924182653427124, -1.653160810470581, 0.8789411783218384, 0.09191270172595978, 0.08473234623670578, -0.3352673053741455, 0.6562419533729553, -0.0439629964530468, -1.5040855407714844, -1.2403703927993774, -2.2243261337280273, -1.9393912553787231, -0.10894718766212463, 1.798661708831787, 2.2870872020721436, 1.3286734819412231, 0.47421616315841675, 0.7537203431129456, -1.3215184211730957, 0.4610652029514313, 0.120980404317379, -0.9196357131004333, 1.1008886098861694, -0.023558981716632843, -0.33528760075569153, 1.1506961584091187, -0.5468580722808838, -0.6317218542098999, -0.5421469807624817, 0.7480573654174805, -0.19892291724681854, 0.8121473789215088, -0.015740135684609413, -0.20431570708751678, 0.5208083987236023, 0.5879275798797607, 0.37034735083580017, -0.9713485240936279, 0.7404657006263733, 1.7312270402908325, 0.2783231735229492, -0.21934042870998383, 0.67764812707901, 0.42027556896209717, 0.011324948631227016, 1.3790128231048584, 1.199615716934204, -0.6728701591491699, -0.7612230777740479, 0.6996892690658569, 0.1947343796491623, -0.11165227741003036, 0.7181466221809387, -0.536177933216095, 0.1978117674589157, -0.10899566113948822, 0.28442540764808655, 0.1901901811361313, -0.8358999490737915, 1.0120781660079956, -0.03266850486397743, 0.395323246717453, 0.724564790725708, -1.4788326025009155, -1.0742071866989136, -0.10431648790836334, 1.8848005533218384, 1.583187222480774, -0.5712703466415405, 0.3527471125125885, 0.14175933599472046, -0.5981260538101196, -0.7332779765129089, -0.33501332998275757, -1.4648007154464722, 0.27121827006340027, -0.557052493095398, -1.3942221403121948, -0.18854498863220215, 0.9820349812507629, -0.39964962005615234, 0.06267049908638, -0.4729765057563782, 1.3941328525543213, -0.3498889207839966, -0.9852607846260071, 0.9124800562858582, 1.9229408502578735, 0.5981247425079346, -1.3240883350372314, 0.8875723481178284, -1.8023912906646729, 0.058220960199832916, -0.6401271224021912, 0.6528476476669312, 0.5435072779655457, 1.5096635818481445, -0.14437665045261383, 1.740875005722046, 2.0911405086517334, -0.13950499892234802, -1.5750077962875366, -0.1548669934272766, -0.02710164524614811, 0.9930633902549744, -1.7390680313110352, 1.554893970489502, 0.5582031011581421, 1.4335390329360962, -0.6269014477729797, 0.547672688961029, 0.8683976531028748, 0.7574360370635986, -1.1423814296722412, -3.023632287979126, 1.0602034330368042, 1.3118910789489746, -1.024818778038025, -2.624298095703125, 0.07024577260017395, -0.04408826306462288, -0.567263126373291, 0.47355833649635315, -0.5331898927688599, 0.17679211497306824, -1.6647543907165527, -1.0879864692687988, -1.4686678647994995, -1.8075345754623413, -0.8204652070999146, -0.5000324845314026, -0.5942423343658447, 0.25559988617897034, -2.361877918243408, -0.9186246991157532, -1.5680913925170898, -0.8503826260566711, 0.4579164683818817, 0.11031205952167511, -0.24566231667995453, 1.8655767440795898, 0.07860836386680603, -0.8186231851577759, 0.42051270604133606, 0.49421823024749756, -0.3922417163848877, 0.8937388062477112, 0.828564465045929, 0.10333125293254852, -1.812193512916565, -0.8524054288864136, -0.26332888007164, 0.03378647193312645, -0.20887787640094757, 1.0855555534362793, 0.07417749613523483, -2.02063250541687, -0.44199249148368835, -0.28248530626296997, 0.26545509696006775, 0.6332933306694031, -0.5101439952850342, -0.8355097770690918, -0.905256450176239, 1.8837026357650757, -1.0421416759490967, -0.9798193573951721, 0.3784632682800293, 0.4161500632762909, -1.213224172592163, -1.5355727672576904, 1.0979218482971191, -0.654373288154602, -0.12574590742588043, 0.11617663502693176, 0.011995933018624783, 0.9226637482643127, 0.2301388382911682, 0.674088180065155, -0.016787342727184296, -0.08820203691720963, 3.1873619556427, 1.5014879703521729, -0.35851016640663147, -0.26154670119285583, -2.7630832195281982, -0.8186841607093811, 0.7534160017967224, -0.34990933537483215, -0.9758512377738953, -0.6146430969238281, -0.17098140716552734, -0.2800523042678833, -1.3667943477630615, -1.3045989274978638, -1.6763861179351807, -1.0633114576339722, -0.8199712634086609, 0.6859561204910278, 0.9306933879852295, -1.0908912420272827, 0.8800855278968811, -0.7371138334274292, 0.3064427077770233, -0.19573266804218292, 0.2491283267736435, -0.06589788943529129, -2.74979829788208, -1.0796817541122437, 2.5533862113952637, 1.1320642232894897, -1.8291529417037964, 0.8903322815895081, 2.797473669052124, -1.0726642608642578, -1.1004196405410767, -1.2280147075653076, 0.34919971227645874, 0.8711294531822205, -0.5613874197006226, 1.7638511657714844, 2.6525931358337402, 1.0882617235183716, -0.18943530321121216, -0.13161134719848633, 0.9814509153366089, 0.7899640202522278, -0.8829993009567261, 0.5227462649345398, -0.8049383163452148, 0.48704585433006287, 0.387190580368042, 0.9966776371002197, 0.0398738756775856, -0.8985860347747803, 0.2435232251882553, 0.9678515791893005, -1.7659766674041748, -0.4802946448326111, 0.849565863609314, -0.34831881523132324, 1.1311100721359253, -0.36374998092651367, -1.5867706537246704, 0.6475204229354858, -0.34577587246894836, 0.8178255558013916, 0.7922643423080444, -0.6756525635719299, -0.995785117149353, 0.8156706690788269, 1.646633505821228, -0.8684008717536926, -0.3337424397468567, 0.24517324566841125, 1.5434203147888184, -0.11051125824451447, 0.7668682932853699, 0.5719707608222961, -0.24384529888629913, 0.3529928922653198, -0.016306651756167412, -0.5125163793563843, -0.17643393576145172, 0.9791708588600159, 0.48600679636001587, -0.8757725954055786, -1.2937513589859009, 2.0300965309143066, 2.062396287918091, -1.2243839502334595, -1.877354383468628, 0.5078569054603577, 1.2364355325698853, -0.6753401160240173, 1.0625767707824707, 0.8849847316741943, 1.2198302745819092, -1.8845922946929932, 0.2728787958621979, -0.5400642156600952, -0.05726555362343788, -0.5522294640541077, 0.4598604440689087, 0.016822675243020058, 0.8199974894523621, 0.12318162620067596, -1.4334137439727783, -0.6659080386161804, 1.2125855684280396, 0.6104195713996887, 0.5141969323158264, -1.4284679889678955, 0.8431618213653564, 0.5942990183830261, 0.5801973938941956, 0.4508373439311981, 0.02120261639356613, -1.8237417936325073, 0.36157920956611633, 0.05009079724550247, 0.9591216444969177, 1.0166329145431519, -0.5965908765792847, -1.526330828666687, 1.0560599565505981, 1.9268995523452759, -1.570486307144165, -2.378887414932251, -0.5636863708496094, -0.032382797449827194, 1.659374713897705, -0.8694242835044861, 0.5442964434623718, 0.2639044523239136, -0.4847487807273865, 2.3764808177948, 0.34509700536727905, 0.46572378277778625, -0.2556964159011841, -0.20604091882705688, -1.9538815021514893, 0.6666141748428345, 0.42856794595718384, -0.8860538601875305, -0.9214760065078735, -0.2368575930595398, 0.21019193530082703, 0.49492549896240234, -0.18270990252494812, 0.12265077233314514, -0.17727798223495483, -0.668476402759552, -1.0732403993606567, 0.8703846335411072, 1.1073315143585205, 1.6339138746261597, -0.09518502652645111, 0.08010563999414444, -1.5460304021835327, 1.9511393308639526, -0.11704789102077484, 1.5208967924118042, -1.03775155544281, -0.48239317536354065, 0.039551325142383575, -0.4038962125778198, 0.20022188127040863, 0.7525282502174377, 1.2971912622451782, 0.40982186794281006, -0.6474066376686096, 0.2344254106283188, -0.9996204376220703, -1.3679035902023315, 0.5414385199546814, 0.4049404263496399, 1.879442572593689, -1.5874409675598145, 1.686141848564148, 0.7422274947166443, 0.31104618310928345, 1.4377892017364502, -1.4594011306762695, 1.4489312171936035, 0.26196566224098206, -0.5905067324638367, 0.5104670524597168, -3.0121309757232666, 0.9862467646598816, -0.021770142018795013, 0.9047086834907532, 1.4483141899108887, -0.12229011207818985, 0.3921491205692291, 0.33826932311058044, 2.70982027053833, 0.4271456003189087, 0.5551141500473022, 0.23358336091041565, 0.4287177622318268, 0.44906240701675415, -0.7504926919937134, -2.8131942749023438, 0.5812124609947205, 0.08019186556339264, 1.5837854146957397, 0.1705075353384018, -0.775012731552124, -0.6465869545936584, 0.5288921594619751, 1.0800553560256958, -1.5859594345092773, -0.817516565322876, -0.29511621594429016, -0.4241887032985687, 0.5514153838157654, 0.6455872058868408, 1.8543164730072021, -0.35544535517692566, 1.3485528230667114, -0.3535281717777252, -1.3992623090744019, -0.8117607831954956, 0.43454188108444214, 1.0625001192092896, 0.7566335797309875, 0.5742968320846558, -0.38413146138191223, -1.2025492191314697, -0.8737190365791321, -1.0769734382629395, 0.3937142789363861, -2.196984052658081, 1.2720146179199219, 0.07912710309028625, 0.29862579703330994, -0.1991659700870514, -0.15021593868732452, 0.06019241735339165, 1.1240792274475098, 1.2383919954299927, -2.1221506595611572, -0.8294493556022644, -1.3617594242095947, -0.7498956322669983, -0.17401544749736786, -0.8120313286781311, 1.2370609045028687, -1.946742057800293, -0.2914017140865326, -1.3074707984924316, 1.0561891794204712, -0.25146007537841797, -0.5005087852478027, 0.3470454216003418, 0.5835900902748108, -0.6876113414764404, 0.4476785659790039, -1.0158600807189941, -0.6121467351913452, -2.043618679046631, 0.8257550597190857, -1.4998711347579956, 2.2985949516296387, 0.6378998160362244, -0.5100966691970825, -0.3020640015602112, 0.4272266626358032, -0.07100407034158707, 0.0648968443274498, 1.7932943105697632, 0.29757413268089294, 1.6995885372161865, -2.97125506401062, -0.9660574197769165, 0.6724646687507629, 0.8885934948921204, -0.2827633321285248, 1.021067500114441, 1.1392383575439453, 0.35912731289863586, -0.31822630763053894, 1.2822933197021484, 0.6171690821647644, 2.3324897289276123, -1.315466284751892, 0.9088564515113831, 0.48690953850746155, 0.05427729710936546, 0.7482836842536926, 0.08296801149845123, -0.6176902055740356, -1.0975725650787354, 0.6693191528320312, -1.9725028276443481, -0.30156081914901733, -0.4183388650417328, 0.09341540187597275, -0.8098875880241394, 0.08519833534955978, -0.7125077843666077, -2.098665237426758, -1.340131402015686, 0.40313205122947693, 0.86199951171875, -2.629337787628174, 0.5394948124885559, -0.23796024918556213, -1.7168766260147095, 0.8870434761047363, -0.341335654258728, -1.5023850202560425, -0.31379935145378113, -1.6061314344406128, 1.7430975437164307, 0.00017613853560760617, 0.2708771824836731, -0.6521514058113098, -0.957858681678772, -0.3048776090145111, -0.28780823945999146, 1.520934820175171, 1.215153455734253, -0.5366827845573425, 1.6863298416137695, -0.004544130060821772, -0.036361854523420334, 0.18555717170238495, 0.09183841943740845, -0.5572168231010437, 1.7695064544677734, -0.3967791497707367, 0.607943594455719, 0.0226224847137928, -0.11345823854207993, 0.5728527903556824, 1.0117615461349487, 0.18335892260074615, -0.881329357624054, 0.09578634798526764, 0.44701048731803894, 0.05608534812927246, 1.5261482000350952, 0.2596197724342346, -0.7826589941978455, 1.583983302116394, 0.5192088484764099, 1.9684789180755615, -1.6757076978683472, 0.45884954929351807, 0.08293543756008148, 1.107521414756775, -0.17992734909057617, -0.0809159204363823, -0.5486364960670471, 1.2324389219284058, 1.3516318798065186, 0.14528299868106842, 0.05187186971306801, -0.17134180665016174, 0.3495630621910095, -0.7187727093696594, -0.3527957499027252, 1.2489384412765503, -0.875653862953186, -0.40565264225006104, 0.2251836061477661, -0.4751400649547577, -1.0411771535873413, -2.5611319541931152, -1.3543645143508911, 1.6103501319885254, 1.6770634651184082, 0.17313286662101746, -0.07132947444915771, -0.5160271525382996, 0.9442589282989502, -0.04248267412185669, -0.7129702568054199, -0.29739540815353394, -0.49190202355384827, 0.0018870464991778135, -0.1982094794511795, -1.4634637832641602, -0.1905258446931839, 1.8178354501724243, 0.03177650272846222, 0.2573449909687042, -1.1398000717163086, -2.1073920726776123, -0.18662472069263458, -0.37959542870521545, -0.4000737965106964, 0.5492790341377258, 0.547432541847229, -0.20927399396896362, -0.432855486869812, 0.6594722867012024, 1.415682315826416, -0.3611992299556732, 0.50111985206604, -1.7031562328338623, -0.7050609588623047, -1.814765214920044, 0.8262677788734436, 0.427245169878006, 0.2362724393606186, -1.2050715684890747, 0.15079139173030853, -0.630036473274231, -0.15933489799499512, 0.5617012977600098, -0.48732441663742065, 1.2018431425094604, -0.8632159233093262, 0.13902662694454193, -0.6286166310310364, 0.8593059182167053, -1.1793075799942017, -0.0017393749440088868, 0.12904572486877441, 0.036720048636198044, -1.194754719734192, -0.8778355717658997, 1.6015169620513916, -2.2770540714263916, -0.6935442090034485, 2.0665969848632812, -0.3110836148262024, 0.37891995906829834, 0.5129480361938477, 1.0496710538864136, -0.16826486587524414, -0.06768542528152466, -0.38495761156082153, -1.148364782333374, -0.03487805649638176, 0.8674243688583374, 0.27482834458351135, -0.5969306230545044, -0.5391346216201782, -0.28997039794921875, -1.172318935394287, -1.000313639640808, -0.24089209735393524, 1.002629280090332, 1.8011285066604614, 0.19252370297908783, -0.7454013228416443, 0.8908112645149231, 1.7136791944503784, -0.010891996324062347, -1.0387681722640991, 0.46622464060783386, 1.4807618856430054, -1.0057204961776733, -0.772781491279602, 0.7627590298652649, 0.9789528250694275, 0.7517682313919067, -1.2447307109832764, -0.8898921012878418, 1.2150115966796875, -0.5142170190811157, 2.2211403846740723, -1.0908058881759644, -1.5053073167800903, -1.0318922996520996, 0.3491758406162262, -0.6653367280960083, 0.1471964716911316, -1.1731070280075073, -0.27952781319618225, -0.5720765590667725, 0.4910784065723419, 1.4725590944290161, -0.012309049256145954, 1.6379963159561157, 0.7347962260246277, 0.758620023727417, 0.03304809704422951, 0.5337004065513611, 0.8673864603042603, -0.7567200064659119, 0.19734911620616913, 1.0351046323776245, -2.110926628112793, 1.6424603462219238, -0.4976000487804413, -0.7973206639289856, 0.5829237699508667, 0.8403589725494385, -0.4266032874584198, 1.28932523727417, 1.0251457691192627, 1.2684584856033325, 0.16484493017196655, -0.2596849501132965, -1.777059555053711, -0.95025634765625, 1.43221116065979, -0.964673638343811, 0.9347403049468994, 0.7698957920074463, -0.28790283203125, 0.05853165313601494, -0.5344265699386597, -1.0292763710021973, 0.3211336135864258, 0.43854430317878723, 0.18404948711395264, 1.0244399309158325, -1.4244093894958496, -1.3405500650405884, -0.03254639357328415, 0.19492436945438385, -1.1707994937896729, -0.753931999206543, 1.3019226789474487, -1.0493613481521606, 0.7233384251594543, 0.517725944519043, 0.42599061131477356, 1.7731999158859253, 0.4995504319667816, 0.1424192637205124, -0.44563600420951843, 0.8100824356079102, 0.4006313979625702, -1.3007198572158813, -0.42406484484672546, -2.3218178749084473, -0.04252740740776062, -0.9520106911659241, 0.45670071244239807, -2.9140281677246094, -0.2837013006210327, -0.2151675671339035, 1.3661670684814453, -0.535663366317749, 1.460194706916809, -0.3511468470096588, -1.0329399108886719, 0.133297398686409, -1.030542016029358, -0.8667909502983093, 0.8347142338752747, 0.23100508749485016, -0.2029881775379181, 0.8901842832565308, -0.027692362666130066, -0.7128937244415283, 1.320441484451294, 0.9009165167808533, 0.7840867638587952, 1.700610876083374, 0.9033493399620056, -0.7180024981498718, -1.3931742906570435, -0.8490121364593506, -1.5012277364730835, -0.09286174178123474, 0.15120790898799896, 0.057842545211315155, -0.18431176245212555, -0.511423647403717, 0.7180240154266357, -0.241222083568573, -1.0372542142868042, -0.322317510843277, -2.230609893798828, -0.7782315015792847, 0.8266834020614624, 1.4985167980194092, 0.24925728142261505, -0.3823336362838745, -0.9270389080047607, -1.3306472301483154, -0.7501299977302551, 1.8827691078186035, 0.3245971202850342, -0.24760808050632477, -0.9311960935592651, -0.8507178425788879, -0.3345405161380768, 0.9643553495407104, 0.3867676556110382, 1.8094518184661865, 0.4833623766899109, -0.2447044402360916, 0.24632765352725983], [0.8893489837646484, 0.15559369325637817, 0.9532855749130249, 0.638704240322113, 1.0219870805740356, -0.8531901240348816, -0.3162374496459961, -0.42113977670669556, 0.0008179505239240825, 0.24989908933639526, 0.06656362116336823, -0.06369805335998535, -1.0483936071395874, -1.0705859661102295, -0.7609732747077942, -0.7317548394203186, 0.9951313734054565, -0.6694193482398987, -0.43658146262168884, 0.7917978763580322, -0.160880908370018, 0.45095300674438477, 0.33016541600227356, 1.5482699871063232, -0.9666051864624023, 0.6166700720787048, -1.2867722511291504, -1.492270588874817, 0.17069830000400543, 2.2247209548950195, 1.5552517175674438, -0.34294623136520386, -1.4077839851379395, 0.5559893846511841, 0.7475802302360535, 0.5514954924583435, 1.0132665634155273, 0.38288071751594543, -0.25512009859085083, -1.4752535820007324, -0.6299599409103394, -0.16062739491462708, 0.6344116926193237, -1.0492109060287476, 1.4476304054260254, 0.23843178153038025, 0.4084795415401459, 0.09500810503959656, -1.262266755104065, 1.770185112953186, -1.6739310026168823, -0.31919601559638977, -2.0634725093841553, -0.1374158412218094, 0.6731351017951965, -1.9101927280426025, -0.9117056727409363, -0.3860248029232025, -2.111910581588745, -1.6824201345443726, -0.7008106112480164, 0.021083058789372444, -0.6691625118255615, 0.9634902477264404, 0.6333982348442078, -0.03714606538414955, 0.553451657295227, -0.25355541706085205, 1.8581668138504028, -0.12145669013261795, -1.7927768230438232, -0.04020436480641365, 1.3378883600234985, 0.30711114406585693, -0.1101938933134079, 1.3411767482757568, 1.1533117294311523, -0.9200854301452637, 0.9909682273864746, 1.3303532600402832, 0.7522832751274109, -0.36893224716186523, 1.2016836404800415, -0.9705439209938049, -0.10877209156751633, -0.06888999044895172, -1.1070811748504639, 1.029558777809143, -0.8691768646240234, -1.4136576652526855, 1.4800903797149658, 1.3106932640075684, -0.07047997415065765, -0.5400931239128113, -0.9717991948127747, 1.2804069519042969, -0.10544468462467194, -0.35209545493125916, -0.5623354911804199, -0.33395928144454956, 1.0048351287841797, 0.8929377198219299, -0.4302785098552704, -0.13845232129096985, 0.37100204825401306, 0.48302820324897766, -0.7047296166419983, 0.30742737650871277, -0.5098802447319031, 1.3314995765686035, -0.5509357452392578, -0.6298122406005859, -2.4266161918640137, -0.15320052206516266, 1.1510547399520874, -0.17854717373847961, -0.9982208609580994, 1.8500299453735352, -0.3150753080844879, 0.4810599982738495, -0.8052583932876587, 0.33619558811187744, -1.6346383094787598, 0.20366689562797546, 1.5944386720657349, 0.045480720698833466, 1.668235182762146, -0.5302495360374451, -1.3062695264816284, -1.2875937223434448, -0.3959469795227051, 1.3472445011138916, -1.046093225479126, -0.20433761179447174, -0.4717649221420288, 0.27199602127075195, 1.4740989208221436, -1.0990955829620361, -0.41483187675476074, 0.4727809429168701, -0.06640761345624924, -0.4091591238975525, 0.8210264444351196, 0.3561722934246063, -1.3513628244400024, 0.24141332507133484, -0.5812827944755554, -0.13328059017658234, 1.6344308853149414, 1.1624752283096313, -0.6044178605079651, 0.5023573637008667, -0.9941309690475464, -0.8033850789070129, -0.38828569650650024, 0.837668240070343, 0.13840463757514954, 1.476904273033142, -0.4418814480304718, -0.1259845495223999, -1.316207766532898, 0.7370789647102356, -0.9395331740379333, 0.055169932544231415, -0.4491601288318634, 0.6677548289299011, -1.0913208723068237, -0.03293924406170845, -0.5220287442207336, 0.5145071148872375, 0.6976012587547302, -0.6982389092445374, 0.4634469151496887, -0.5594741106033325, -1.1726458072662354, -0.743934690952301, 0.34395739436149597, 1.0118162631988525, 1.4337130784988403, -0.082705058157444, -0.2510279417037964, 0.8834314942359924, -0.43572527170181274, -0.7329265475273132, 0.5032098889350891, -0.030072122812271118, 1.5960147380828857, -2.089318037033081, -2.9059367179870605, 0.28975436091423035, -1.782890796661377, -0.24818243086338043, -1.170291543006897, 1.0221331119537354, -0.563974142074585, -0.22793124616146088, -1.2237768173217773, -2.0263333320617676, 0.08210679143667221, 0.31562891602516174, -0.1239808052778244, 1.204364538192749, 0.7143197655677795, -1.4666293859481812, 0.4835571050643921, -0.5686311721801758, 0.24154528975486755, 0.029125381261110306, -1.5711300373077393, 0.08902251720428467, -1.2714359760284424, 1.4241218566894531, -0.7352908253669739, -0.7340415716171265, -0.03903748095035553, -0.42654475569725037, -0.07462894916534424, -1.0957121849060059, -0.6948946714401245, 0.03532792255282402, 2.045464038848877, -0.6938930153846741, 0.8748745322227478, -0.15228436887264252, 1.3337570428848267, 1.0949639081954956, 1.0004736185073853, -0.9377421140670776, -1.5831209421157837, -0.7252787351608276, 0.6686571836471558, -0.9029943346977234, 1.4252021312713623, -1.2456364631652832, 0.5118481516838074, -0.5652053356170654, -2.0555739402770996, -0.29686757922172546, 0.6744588613510132, 0.40070560574531555, -0.9092170000076294, -0.38942471146583557, 1.2000524997711182, 1.0610132217407227, 0.3241012394428253, -0.7535954713821411, 1.3991838693618774, -0.493593692779541, -0.322172075510025, -0.6934340596199036, 1.895156979560852, -0.10916696488857269, 1.3745641708374023, -0.012899326160550117, -0.008819276466965675, -1.369749903678894, 0.05985714867711067, 0.6414411664009094, 0.5768036246299744, -0.20150721073150635, -0.9771310687065125, -0.8963820338249207, 1.9728773832321167, 0.826106607913971, -1.9501069784164429, -0.6222324967384338, 0.49729716777801514, -0.5907728672027588, -1.6805990934371948, 0.816647469997406, 1.9612983465194702, 0.40736863017082214, 0.8494654297828674, -0.8290639519691467, 1.063817024230957, -1.2562403678894043, 0.46280133724212646, -0.34901556372642517, 0.42850032448768616, 1.853275179862976, 1.1271367073059082, -0.4390190839767456, 1.1627466678619385, -1.7173196077346802, 0.8137496113777161, 0.059942349791526794, 0.3306439220905304, 0.8627902865409851, -1.5143529176712036, -0.033146459609270096, 1.4329791069030762, 0.16519615054130554, 0.1401076316833496, 1.9281225204467773, -0.020764518529176712, 0.41768959164619446, -0.6394497156143188, 0.9370629191398621, 0.5484355688095093, 1.6333012580871582, -0.6283282041549683, -0.9960830807685852, -0.5659677386283875, -0.9558935165405273, 1.1236941814422607, 1.8769450187683105, 1.7389942407608032, 1.6290265321731567, 0.12071507424116135, -0.6198928952217102, -0.8609408736228943, 1.0849963426589966, 0.7044486999511719, -0.4581029713153839, -0.3203001320362091, 0.9576126337051392, -0.9079229831695557, 1.3346474170684814, 1.2942150831222534, 0.9547840356826782, 0.12223608046770096, -1.5767041444778442, 0.4521196782588959, -1.0230106115341187, 2.1732864379882812, -0.1813662350177765, -0.41468966007232666, 0.6623515486717224, 0.7421074509620667, -0.9050238728523254, 0.930346667766571, -1.129090428352356, -0.055100347846746445, -0.6387676000595093, 2.2660534381866455, -2.1689493656158447, 0.37894901633262634, 0.44021326303482056, -0.37602654099464417, 0.8812097311019897, -0.04046272858977318, 0.7117900848388672, -1.3519580364227295, 0.3010736107826233, 1.5836087465286255, 1.6326043605804443, 0.5159464478492737, 0.7088121175765991, -0.1332467794418335, 0.10341636091470718, 0.18687179684638977, 1.1185892820358276, -0.22455213963985443, -0.6354383230209351, 1.6113710403442383, 1.152276635169983, -0.3319331109523773, 0.2677118182182312, -0.6402392983436584, 0.5074594616889954, -0.12489902228116989, -1.6406947374343872, 0.4175894856452942, 0.23919837176799774, 1.4191025495529175, 0.13936790823936462, -0.8173190951347351, 0.4123702347278595, -0.4888336956501007, 1.73726487159729, 1.4160045385360718, -1.0068484544754028, 1.7501453161239624, 0.6854535341262817, -0.021333931013941765, 0.21963386237621307, -1.1433004140853882, -0.8089001178741455, 1.3061096668243408, -0.6749663352966309, -0.9190307259559631, 1.0762149095535278, 0.8723868131637573, -0.35821324586868286, -0.876663863658905, -0.583652675151825, 0.39182502031326294, -0.46733492612838745, 0.7955909371376038, -0.7148467898368835, 1.3547099828720093, 0.3233489990234375, 1.1417407989501953, 1.055384635925293, -1.1295506954193115, -1.4587688446044922, 1.1646870374679565, 1.7893818616867065, 0.5767728686332703, 0.4661194682121277, 0.3038395941257477, 0.31415584683418274, 0.3156762719154358, -0.6237136125564575, -2.04947829246521, -1.6609511375427246, -0.39215564727783203, -0.24805080890655518, -0.4391050636768341, 0.685634195804596, -0.834585964679718, 0.887144923210144, 1.4531430006027222, 0.283429890871048, 0.39635372161865234, -1.302340030670166, 0.21244512498378754, -0.29878485202789307, 1.6475231647491455, -0.2204553782939911, 0.5759599208831787, 0.23781746625900269, -1.3947638273239136, -0.410592645406723, 0.06710582226514816, -0.48140615224838257, -0.4231131672859192, -0.24736525118350983, 0.3309309482574463, 1.097326636314392, 0.13352598249912262, -0.3272302448749542, -0.6194488406181335, 1.4039030075073242, -0.04961134493350983, -1.0418431758880615, 1.2441500425338745, 1.6274161338806152, -1.0924192667007446, 0.19094553589820862, 0.3645937442779541, 0.901493489742279, 0.42436307668685913, -0.8246992826461792, 0.46861422061920166, 0.2786760926246643, 0.1658175140619278, -0.3596189320087433, -0.04152418300509453, 1.6234877109527588, -0.364244669675827, 1.0646692514419556, -0.8653087019920349, -1.7273313999176025, 1.850407361984253, -1.149965524673462, 1.0023164749145508, 0.9470067620277405, -0.5524682402610779, 0.7590875625610352, -0.23423095047473907, 1.0497157573699951, 1.2675483226776123, -0.27345097064971924, 0.38057249784469604, -0.2194739729166031, 1.3808811902999878, -0.8942959308624268, -1.2334091663360596, 1.4045621156692505, 0.39678671956062317, -0.004193175118416548, -2.86064076423645, 0.8372690081596375, 1.2027589082717896, -0.26466426253318787, 0.3167757987976074, -0.7117801308631897, -0.6620122790336609, -0.05002504214644432, -0.04734671115875244, 0.15326784551143646, -0.34804606437683105, -0.8046130537986755, -0.36925166845321655, -0.6188872456550598, 0.6009299755096436, 0.0012333375634625554, -0.9237218499183655, -0.17566634714603424, 1.0286248922348022, 0.3013192415237427, 0.6578823328018188, -0.8079103827476501, -0.014199456200003624, -0.35741230845451355, 0.12730902433395386, -0.194102481007576, -1.4798909425735474, 0.36098790168762207, 0.15524974465370178, 1.106751561164856, 0.003967560827732086, -0.03465474396944046, 0.3966637849807739, 1.0381531715393066, -0.16841629147529602, 0.6373521685600281, -0.1477232128381729, -0.31424480676651, 0.6803494095802307, -0.6260019540786743, -0.5740088820457458, 0.5085625648498535, -0.08084635436534882, -0.5528470277786255, -1.5339844226837158, 0.9959069490432739, -0.4990726709365845, 0.4705316126346588, -1.1735233068466187, -1.3243939876556396, -0.7853416204452515, -0.9231297969818115, 1.0866128206253052, -0.42658770084381104, -2.411956310272217, 0.8183722496032715, 1.055841326713562, 0.8500057458877563, -0.421146422624588, 0.5686438679695129, -0.4563005268573761, -0.07017387449741364, -0.9298604726791382, -1.8095778226852417, 0.13664349913597107, 1.0248678922653198, -1.9729903936386108, -0.353422611951828, 0.9443642497062683, -0.24351713061332703, 2.469163656234741, 0.2984708845615387, 1.5109694004058838, -1.479568362236023, 0.38635390996932983, 0.8515580296516418, -0.49795278906822205, 0.9071056246757507, 0.26289376616477966, -1.401198148727417, 0.3639110326766968, -0.6602045893669128, -0.629126250743866, -1.6209625005722046, 2.453885316848755, -0.5064194798469543, -1.3996905088424683, -0.4106569290161133, 0.12188715487718582, 1.1586107015609741, -0.8062254190444946, -1.132196307182312, 1.3947203159332275, -0.9601513743400574, -0.24415872991085052, 1.248913049697876, 0.007314543705433607, -2.5862128734588623, -0.01915866881608963, -0.022707825526595116, -0.8542375564575195, -1.4517061710357666, -0.43094372749328613, 0.3329795002937317, 0.06830940395593643, 0.4425162672996521, 1.3015930652618408, 1.1641207933425903, 0.5729677081108093, 0.8242462277412415, -0.3695106506347656, -2.2801053524017334, -0.171730637550354, 1.1385548114776611, -1.1620961427688599, 0.626406192779541, -0.49618756771087646, 0.8453544974327087, -0.6755011677742004, -0.6902378797531128, -1.5114761590957642, -1.238778829574585, -0.4678708016872406, -1.8021366596221924, 1.369589924812317, -0.07560531049966812, 0.9458588361740112, -0.6035559773445129, 0.4349993169307709, 0.6739364266395569, 0.4373567998409271, 0.7256230711936951, 0.2646333575248718, -0.2882468104362488, -1.26893949508667, -0.8895449638366699, -0.3766125440597534, -0.09674873948097229, 0.5886800289154053, -0.39802706241607666, 0.8090369701385498, 0.35144859552383423, -1.072837471961975, -0.2736830711364746, -1.3187545537948608, -1.068756341934204, -0.17415501177310944, 0.260063499212265, -0.11198100447654724, -0.47000476717948914, -0.3185606598854065, -0.8431681394577026, -0.4066788852214813, 2.0687289237976074, 0.2711505591869354, -1.9579445123672485, -1.075190544128418, 0.9181632399559021, -1.5653542280197144, -0.14204582571983337, 1.7429618835449219, -0.1911378651857376, 0.4871656000614166, 0.09801697731018066, -0.846275269985199, 1.3442105054855347, -0.4451044797897339, 1.1930162906646729, -1.4092973470687866, 0.17644761502742767, 1.47214937210083, 0.7202909588813782, -0.6945894956588745, -0.7533285617828369, 1.0080952644348145, 1.187495231628418, 0.37781980633735657, -1.7775661945343018, 0.9806830883026123, 2.1965959072113037, 0.49401697516441345, -1.740050196647644, -1.9020121097564697, 0.13613061606884003, 0.1635597050189972, -1.6070481538772583, -1.57585871219635, 0.6684826612472534, -0.2359996736049652, 0.6907364726066589, -0.8321084380149841, -0.2195291519165039, -0.4339107275009155, 0.5658369660377502, 2.2520196437835693, 0.0738578662276268, 0.18852777779102325, 0.7725536823272705, 0.059030093252658844, -0.6160983443260193, 0.12117034196853638, -0.7597494125366211, 0.9799633622169495, -1.1431952714920044, 0.47183752059936523, 0.0862109586596489, -0.6162278056144714, -0.11722469329833984, -0.61776202917099, 0.9345230460166931, 0.06085528805851936, -1.4126583337783813, 0.48712053894996643, -0.08574935048818588, -1.8016637563705444, -1.6464998722076416, 1.1172759532928467, -0.8870353698730469, -0.32910168170928955, 0.5005953907966614, -1.2827403545379639, -0.5848491787910461, -0.6972424983978271, -1.0842139720916748, -0.571546196937561, 0.2732192575931549, -1.132301688194275, -0.13160641491413116, -0.041324056684970856, 0.8564432859420776, -0.02753492072224617, -0.23107482492923737, 0.5181928873062134, 0.3622864782810211, 0.5048407316207886, 1.0502057075500488, -0.815879762172699, 0.4531157612800598, 0.0014811510918661952, -1.0134453773498535, 0.8385912775993347, 1.5005077123641968, 0.6084892153739929, -1.7855366468429565, 1.7991013526916504, 0.5920233130455017, 0.7708218097686768, -1.1749951839447021, 0.32176876068115234, 1.2176711559295654, -0.9063076972961426, -0.4035940170288086, -2.408651351928711, 1.3349159955978394, 1.713362455368042, 1.0888680219650269, -1.414018988609314, 0.6039206385612488, -0.3037649095058441, -1.8177177906036377, -0.8264444470405579, 0.8408046364784241, -1.0133486986160278, 0.12162350118160248, 1.2013206481933594, 0.2886587977409363, -0.8716387152671814, -0.15070563554763794, -0.43582242727279663, -1.1686877012252808, -0.36938127875328064, 0.8399848937988281, -0.6851648092269897, 1.3787250518798828, 1.4485273361206055, -0.5881469249725342, 0.12648241221904755, 1.3206477165222168, 0.9335799813270569, -0.02970162034034729, 0.2248820811510086, -0.22568784654140472, -0.45399197936058044, -0.058496396988630295, 1.085605263710022, 0.048564840108156204, 0.6785559058189392, 0.7633587121963501, -0.08840077370405197, -0.4696447551250458, 0.29957279562950134, -0.9968749284744263, 0.7426887154579163, -0.22771291434764862, -1.4379513263702393, -0.33600133657455444, -1.6371142864227295, -0.213825523853302, 1.14036226272583, 1.5936816930770874, 0.021696502342820168, -0.4610954821109772, -0.7327889800071716, 0.6268059015274048, 0.1280675232410431, 0.4912046492099762, -0.42118901014328003, -0.4220081865787506, -0.39053329825401306, -0.5336024165153503, 1.4688233137130737, -0.2690778374671936, 0.10794682055711746, 1.6432377099990845, -1.460132122039795, 0.38496047258377075, 0.04505527764558792, 0.296792209148407, -0.9738928079605103, -0.16252650320529938, -0.6680938601493835, 1.541739583015442, -0.7827043533325195, 0.134311705827713, 0.296099454164505, -0.22313347458839417, 0.8949829936027527, -1.1263643503189087, -0.7466985583305359, 0.4610041379928589, -0.004559234715998173, -0.23520204424858093, -0.5330246686935425, 0.8827319145202637, 1.2857816219329834, 0.08222571760416031, -0.09779711812734604, -0.23515260219573975, -0.2916783392429352, 1.9158343076705933, 0.4543007016181946, 1.3150362968444824, -0.6457900404930115, 0.7055777311325073, -0.21141457557678223, -0.1997203230857849, 0.26317131519317627, -0.4246659576892853, 0.7075518369674683, -2.379399061203003, -1.5253639221191406, -0.1862526684999466, -1.227222204208374, 1.598136067390442, 0.361646831035614, -1.5977619886398315, 0.5935990214347839, 1.780019760131836, -0.20186029374599457, 0.1690872311592102, 0.3446178734302521, -1.1512935161590576, 0.45261135697364807, -0.5522788166999817, -0.006241438910365105, 0.36090996861457825, -1.448960781097412, -0.6422763466835022, -0.026445189490914345, 0.4553539752960205, 1.7707860469818115, 0.9271524548530579, -0.6097609996795654, 1.5319663286209106, -0.6844172477722168, 1.6417683362960815, -1.7569941282272339, -0.3784109950065613, 1.3031738996505737, -1.5854904651641846, -2.264111280441284, 1.2650066614151, -0.3272346258163452, 0.9025905132293701, -0.43085330724716187, 1.3682122230529785, 2.30863356590271, 0.7841261029243469, -0.9586005210876465, -0.310547411441803, -0.5509255528450012, -0.021156935021281242, 0.4560758173465729, -0.37632080912590027, -0.7271192669868469, 1.174673318862915, 0.22328084707260132, 1.4489799737930298, -0.9835121035575867, 1.5408847332000732, 0.07550261169672012, -0.6448619961738586, 0.06367002427577972, -1.0998731851577759, -1.275716781616211, -1.2025408744812012, 0.6622570157051086, -1.126190423965454, -0.06942912936210632, 0.17846357822418213, -1.5971040725708008, 0.5545268058776855, -1.2929162979125977, 1.8170456886291504, -0.4157670736312866, 1.5833371877670288, 0.9139670729637146, 0.23237662017345428, -0.8756034970283508, 0.0017870597075670958, 0.8387735486030579, -0.014401113614439964, 0.10554665327072144, 0.08762677013874054, 0.09334404021501541, 2.2366292476654053, 0.9469573497772217, -0.03171543404459953, 0.5998403429985046, 0.5002793669700623, 0.384071409702301, 1.627465009689331, -0.407040536403656, 0.7329308986663818, 0.537267804145813, 0.17957834899425507, -0.7253226637840271, 1.7614502906799316, 0.8112240433692932, 1.3355698585510254, 0.48769351840019226, 0.36633187532424927, 0.057538505643606186, -0.6364378929138184, -1.8784000873565674, -1.6228797435760498, 0.9217084646224976, -0.5868649482727051, 0.12367870658636093, 0.729232907295227, -0.26214781403541565, -0.9549933075904846, -0.9674780964851379, -0.5430068969726562, -1.267194151878357, -0.6718016266822815, 0.20139093697071075, -0.014255517162382603, -0.5397071242332458, -0.9398757219314575, -0.6240309476852417, 0.7226833701133728, 0.15261080861091614, 0.6606891751289368, 2.1143593788146973, -0.5842588543891907, -0.6557784080505371, 0.6054374575614929, -1.0069384574890137, -0.13090236485004425, -0.41022562980651855, -0.5714211463928223, -0.1739046424627304, -0.08973930776119232, 2.5498924255371094, 1.6900784969329834, -0.42611101269721985, 0.10486975312232971, -0.6722941398620605, 0.2071312963962555, 0.22452130913734436, -0.9934946894645691, 0.696973979473114, 0.9959542751312256, -0.33484312891960144, -1.8655940294265747, -0.175608828663826, 1.453181266784668, -0.05983706936240196, 0.7900193929672241, 0.16178911924362183, 0.24890387058258057, 0.35231679677963257, -0.510332465171814, 0.6793636083602905, 0.38870933651924133, -0.004874637350440025, -0.10136209428310394, -0.5875624418258667, 1.2588053941726685, 0.21309983730316162, 0.6395609378814697, 0.05369439348578453, -0.6469739079475403, -1.017188310623169, 1.5248262882232666, 0.9177829027175903, -0.24939103424549103, -1.7933933734893799, 0.5806452631950378, 0.584466278553009, -0.5310268998146057, 0.3817123472690582, 0.00683584064245224, -0.8949990272521973, 0.2626420855522156, 0.9742196202278137, -0.2341984361410141, -0.2099805772304535, -0.9160048365592957, -1.1067479848861694, 0.668129563331604, -0.47448521852493286, -0.8589116930961609, 0.5430090427398682, -0.6096816658973694, 0.2553333640098572, -1.1254719495773315], [0.3777589499950409, 0.06258653104305267, -0.695708692073822, 1.6627748012542725, -0.6883028745651245, 1.3721072673797607, 0.850386917591095, -0.818960964679718, -0.20149743556976318, 1.3135793209075928, -0.17503724992275238, -1.408103585243225, 1.3306992053985596, 0.6992471814155579, -0.08837329596281052, -1.0754303932189941, -1.0240864753723145, 1.5744622945785522, -1.269956111907959, 0.9128707051277161, -0.6733980774879456, 1.195202112197876, 0.9741679430007935, -0.27697810530662537, -0.3545531630516052, 1.299951195716858, 1.3261185884475708, 3.2254867553710938, -2.1039438247680664, -0.998136579990387, 0.7497047781944275, -0.9163962602615356, -1.2616593837738037, -1.4050238132476807, -0.013596647419035435, 1.5041604042053223, -0.17687830328941345, 0.4454765021800995, 0.2859637141227722, -1.7895114421844482, 0.48649001121520996, -0.4119691550731659, -0.31454843282699585, 1.7022244930267334, 0.7589757442474365, 0.19594205915927887, -1.081047534942627, -0.28321337699890137, -0.9783759713172913, -0.2384425401687622, 0.8568532466888428, 0.09171908348798752, -0.8494375944137573, 0.2889663577079773, -1.5663468837738037, -0.2364378273487091, 0.8794224858283997, 0.4620170593261719, -0.42421481013298035, -0.38074371218681335, -1.0680180788040161, 0.5991125702857971, -1.4848897457122803, 0.6723499894142151, 0.05535924434661865, 0.28647127747535706, -0.32812175154685974, 0.8183506727218628, 0.7964516878128052, 0.22343838214874268, 0.26571741700172424, 0.9651152491569519, 0.9239421486854553, -0.6457239985466003, 0.02770305424928665, -1.4484840631484985, 0.4641319215297699, 1.3335696458816528, -0.4746023714542389, -0.5514082908630371, 0.5382747650146484, 1.136925220489502, -0.5828882455825806, -0.16359710693359375, 0.32138943672180176, -0.5065733194351196, -1.0848009586334229, 0.37880828976631165, -0.9948378801345825, 0.9257183074951172, 1.0687999725341797, -0.7620264887809753, 0.11565215140581131, 0.6456112861633301, 0.003915951121598482, -0.047814033925533295, 0.5712251663208008, 2.19088077545166, 0.7471486330032349, 0.8475247621536255, -0.6043049693107605, -0.06751503050327301, -0.6471095085144043, -2.99748158454895, 0.09809450060129166, 0.25149431824684143, -0.45680612325668335, -0.2259976714849472, -0.7079541683197021, -0.738388180732727, 0.26416561007499695, 0.3570207953453064, 0.4245726466178894, -0.34357792139053345, -0.6634505391120911, -1.142197847366333, 1.6037611961364746, -0.5756776928901672, -0.1969238519668579, 0.32143881916999817, -2.196554183959961, 0.08822394162416458, 1.0566025972366333, 0.07355085015296936, -0.4703666865825653, 0.12382535636425018, -0.5646469593048096, 0.05780757591128349, -1.6681294441223145, -1.5445283651351929, -0.35648804903030396, 0.07682168483734131, -0.4228613078594208, -1.3330023288726807, -0.4401605427265167, -0.4427315592765808, -1.8809133768081665, 0.45930466055870056, -0.5592933893203735, -0.15558938682079315, -0.6364084482192993, -0.6859750151634216, 0.3067642152309418, 0.49030032753944397, -0.6907861828804016, -1.0022473335266113, 1.5126149654388428, -0.12749788165092468, -1.6945942640304565, -0.07712822407484055, 1.068579912185669, 0.0249099750071764, -0.07262753695249557, 0.18895794451236725, 0.28380587697029114, 0.7694535255432129, -1.2019699811935425, 0.4044361412525177, 0.7896705865859985, -0.9700336456298828, -1.3423179388046265, -0.14704692363739014, 0.8786119818687439, -1.631941318511963, -0.14761878550052643, 0.8228174448013306, -1.6508259773254395, 1.362518072128296, -0.17462164163589478, -0.8288095593452454, 1.470057725906372, 0.4656227231025696, 0.6624540090560913, 0.6886785626411438, -0.48696500062942505, 0.008323430083692074, -0.5344892740249634, -1.8060617446899414, -0.890058159828186, 0.9281018376350403, 2.2086164951324463, 1.5036368370056152, 0.31871500611305237, -0.7437292337417603, -0.36843109130859375, 1.0033177137374878, -0.508159339427948, -0.4354137182235718, -0.03879166021943092, -0.7792671322822571, -0.15405836701393127, 0.007017816416919231, 1.2630183696746826, 0.47429123520851135, -1.0363376140594482, 0.28819766640663147, -1.4129877090454102, 1.1624864339828491, 0.6377509236335754, 0.8318082094192505, 0.45608508586883545, 0.7285357713699341, 0.7142336368560791, 0.23692792654037476, -0.8879398107528687, 0.08670589327812195, -2.290158271789551, -0.42315399646759033, -0.879625678062439, -1.1374400854110718, 0.8487400412559509, -2.0932364463806152, 1.1188970804214478, 2.8721423149108887, 1.5181766748428345, -0.036856360733509064, 0.7465159296989441, -0.7712705135345459, 1.1443958282470703, 0.04872975870966911, 0.9012510776519775, 1.7253258228302002, -0.30104175209999084, -0.4010622501373291, 1.6385867595672607, 0.6049880385398865, -0.9316867589950562, 1.3972030878067017, -1.227165937423706, 0.787543535232544, -1.2334636449813843, -0.023279784247279167, -1.6198776960372925, 0.5609673261642456, 1.3864985704421997, -0.6123051047325134, -0.2365586757659912, -0.3211665451526642, 0.2902032732963562, -1.276986837387085, 0.8112168312072754, -0.3251607120037079, -1.1579283475875854, 1.2914843559265137, -1.4956941604614258, 1.421221137046814, -0.11259917169809341, -0.4156803488731384, -1.2119336128234863, -0.7617514729499817, -0.7773157954216003, -1.132198452949524, -0.4836583435535431, 0.14156924188137054, 1.129356026649475, -2.3238511085510254, -0.9831873774528503, -0.18155480921268463, 0.49274271726608276, 0.3799891471862793, 0.5499352216720581, -0.9764860272407532, -0.8042727112770081, 0.8056177496910095, 1.0286918878555298, -0.2873384952545166, 0.5874477028846741, -1.3359079360961914, 0.24328310787677765, 0.529800295829773, -0.38812193274497986, 2.697082281112671, 0.5666472911834717, 1.044291615486145, -0.16742639243602753, -1.2509218454360962, -0.030698098242282867, -0.25923267006874084, 0.4865227937698364, -0.9617146849632263, 0.2947951853275299, 0.4070988893508911, 0.5520862340927124, 0.2767235040664673, -0.25348371267318726, 0.8893678188323975, -0.49147483706474304, -0.5341799259185791, -0.6171811819076538, -1.1052651405334473, 1.00263249874115, -1.277819275856018, -1.3277851343154907, -0.9930272102355957, -0.7351406812667847, 0.03734526038169861, -0.8093864917755127, 0.6671777963638306, -1.0803413391113281, 0.5070992112159729, 1.3192802667617798, -0.39230379462242126, -2.6665611267089844, -2.1977272033691406, -0.09437486529350281, 1.1465519666671753, -0.011832315474748611, 0.5285620093345642, -2.1338062286376953, 0.9260376691818237, -0.14381931722164154, -1.925286054611206, 1.1013741493225098, 0.29036930203437805, -1.0934717655181885, -0.7287799715995789, -0.3246835768222809, -0.8073686361312866, 0.48169630765914917, -0.10798284411430359, 0.015277745202183723, -0.1376291811466217, -1.0267504453659058, 0.8576993346214294, -0.36315757036209106, 0.331756591796875, 0.012063615024089813, -2.350733995437622, -0.3040810823440552, 0.04250214993953705, -1.408835768699646, -1.5109643936157227, 0.3606606423854828, -1.4970154762268066, 0.8479454517364502, -1.698587417602539, -1.2433760166168213, -0.2748022675514221, 0.6805523037910461, 0.26579973101615906, -0.617192804813385, 0.8394040465354919, 0.7016069293022156, -1.0413578748703003, -0.17175151407718658, 0.016139637678861618, -0.3929236829280853, 0.20455290377140045, 0.06373818963766098, -1.0208239555358887, -0.8078200221061707, 0.22596514225006104, -1.1204410791397095, 0.8613828420639038, 0.34936845302581787, -0.3047407269477844, -0.07092124223709106, -0.8487991094589233, -1.0285950899124146, 0.1835770606994629, 0.5604608654975891, -0.12774182856082916, -0.48920756578445435, -1.5892386436462402, -0.3584495484828949, 0.5757064819335938, -1.2478467226028442, -0.7473973035812378, -0.5660526156425476, 0.1010410338640213, 0.2550380825996399, 0.7779929637908936, 0.13250324130058289, 1.3464027643203735, -0.2793295979499817, -0.12232217192649841, 0.1268683224916458, 1.6820622682571411, -0.3707502484321594, 0.5658265948295593, 1.0263909101486206, 0.003699560882523656, 0.8180952072143555, -0.6974077820777893, 0.97198486328125, -1.2128368616104126, -0.49158820509910583, -0.8776080012321472, 0.7795112133026123, 0.14791645109653473, 1.5801769495010376, -0.13102227449417114, -0.5678715109825134, -0.9654333591461182, 1.1320037841796875, 0.3685697317123413, 1.1612547636032104, 1.946276068687439, 2.2819032669067383, 1.1950631141662598, -2.1831018924713135, 2.2635836601257324, 0.4715476930141449, -0.48182952404022217, -0.815723180770874, -0.3230040669441223, 1.0318104028701782, -1.1138206720352173, 0.6942121982574463, -0.8002138733863831, -0.7812758684158325, -0.7020620107650757, 0.04643898084759712, 0.5151606798171997, -0.7387146353721619, 0.024483809247612953, 1.3994388580322266, 0.05855577439069748, 1.6634920835494995, -1.5847493410110474, -0.31960195302963257, 0.3751198947429657, 1.0195950269699097, -0.567138671875, 0.1495487093925476, -1.0566599369049072, -0.15512177348136902, 0.007523552514612675, 0.5757823586463928, 0.3912578821182251, -0.3721323311328888, 1.890181064605713, 0.36495235562324524, 0.7951963543891907, -0.034490637481212616, 0.15630978345870972, 0.6978076696395874, -1.902329444885254, -0.47625479102134705, -0.38679414987564087, -0.8669521808624268, 0.5184411406517029, 0.842341959476471, 0.08557749539613724, 0.23271040618419647, 1.6818264722824097, 1.2052546739578247, 0.7703916430473328, -1.6565767526626587, -1.178814172744751, -0.08464693278074265, -2.58064341545105, 0.07295971363782883, 0.37662509083747864, 0.3882305920124054, -0.5180512070655823, 1.5769269466400146, 1.0518733263015747, -1.0784287452697754, 0.7108385562896729, -0.17098361253738403, 0.5813629031181335, 0.4344177544116974, 1.464686393737793, 1.8827143907546997, -0.3196091055870056, -0.49234533309936523, 0.9737237095832825, -0.9721001386642456, -1.3961256742477417, 0.057473745197057724, -0.9823413491249084, 0.24509906768798828, -0.759467601776123, 0.26916101574897766, -0.5070133805274963, -1.280462622642517, 1.1183925867080688, 0.3216203451156616, -0.17343533039093018, -2.2328834533691406, -0.5759223103523254, 0.22990986704826355, -0.10710801184177399, 0.6928749084472656, 0.931574821472168, 1.351051688194275, 1.4411931037902832, 0.48833438754081726, -0.07527845352888107, 0.3707844913005829, 0.2538926303386688, -0.39814651012420654, 0.003555546049028635, 0.31638726592063904, -0.12648239731788635, -0.8742060661315918, 0.27196913957595825, 0.604417085647583, 0.7950952649116516, 0.9055721163749695, 0.05032901465892792, -0.9801292419433594, 0.001739636529237032, -0.2504304051399231, -0.502291738986969, 0.5652589201927185, 0.23610611259937286, -1.5724283456802368, -1.8300501108169556, -0.5965132117271423, 0.11768470704555511, -1.1415480375289917, -0.0865742564201355, -0.19856445491313934, 0.005678272806107998, -0.0924263671040535, 0.915302574634552, 0.1136927679181099, -0.02728560008108616, -0.8987405300140381, 0.9633285403251648, 0.60666424036026, 0.1401224285364151, 0.21648524701595306, 0.02239510603249073, -0.3170478940010071, -1.36604642868042, 0.6393730640411377, 0.9107131958007812, 0.2964033782482147, -2.876631259918213, -0.4176587462425232, 0.7840473651885986, -0.8420780897140503, 1.6168659925460815, 0.7128962278366089, -1.4725819826126099, -0.7036986947059631, -0.5574815273284912, -1.6339799165725708, -0.36831533908843994, -0.46124693751335144, -0.9009546041488647, 0.930688202381134, 0.6629396080970764, -0.47217822074890137, -1.1063238382339478, -1.4917078018188477, -0.36169955134391785, 1.1953033208847046, 0.38745495676994324, 1.5054527521133423, -0.20923392474651337, 0.8925318121910095, -0.24515056610107422, -0.9315056204795837, 1.5990756750106812, 1.655969262123108, 1.7582753896713257, 1.5199209451675415, -1.9596490859985352, 0.9032212495803833, 0.9243226051330566, -1.3951618671417236, -1.9160583019256592, 0.004576630890369415, 0.6437805891036987, -0.12184948474168777, -0.21526826918125153, -0.1832956224679947, 1.157203197479248, -0.4605427384376526, -0.6387475728988647, -0.20896625518798828, -0.5069522261619568, 0.2748439610004425, -0.33485352993011475, 0.3812362551689148, 0.021940505132079124, 1.9125508069992065, -0.5686619877815247, -0.9306972622871399, 0.060448601841926575, 0.36363279819488525, -0.9735159277915955, 1.8853760957717896, 0.9242631793022156, -0.5890610814094543, 0.6142690777778625, 1.2807704210281372, 0.37480178475379944, 0.17240436375141144, 1.1986325979232788, 0.7369639873504639, -0.7269115447998047, -0.7191872000694275, 0.4943925440311432, 1.1312313079833984, 0.6862639784812927, 0.32566285133361816, -0.5240622758865356, 0.5440499186515808, 0.021913956850767136, -0.685411810874939, -0.3666370213031769, 0.9720906019210815, 0.36682894825935364, -0.5276739001274109, 1.1290518045425415, -0.1597057431936264, -1.2952959537506104, 0.5395038723945618, -1.0957354307174683, -0.4011271595954895, 0.8547888398170471, -0.6792475581169128, -0.5145795345306396, -0.06217261403799057, -0.6610673666000366, -0.4185587167739868, 0.9855679869651794, 0.6044508218765259, 0.29983872175216675, 0.06102975085377693, 0.4169212281703949, -0.29284432530403137, 0.5990751385688782, -0.48019200563430786, 0.541928768157959, 0.7858368158340454, 0.4563625752925873, -0.3925577402114868, -0.7969385385513306, 1.3956032991409302, -2.102741241455078, 1.3324054479599, -0.5313189625740051, 0.3529857099056244, 0.6633933782577515, 0.05991455167531967, 1.6647374629974365, -0.6155417561531067, 1.9155855178833008, -1.12447190284729, 0.509036660194397, -0.8695082664489746, -1.400797724723816, -0.9299245476722717, 0.6154340505599976, -0.1061665341258049, 1.175481915473938, 0.6393640041351318, 0.5816953182220459, -1.4969353675842285, 0.7776058912277222, 0.03015938028693199, 0.5900951623916626, 0.5435627102851868, -0.8636648654937744, 0.17364206910133362, -0.9014608860015869, 1.152024269104004, 0.15417979657649994, 0.9640173316001892, -0.9075795412063599, -0.3535751700401306, -0.08069099485874176, 0.4505394995212555, -1.7556829452514648, 0.7019816040992737, -0.7650355696678162, 0.8242113590240479, 0.15337318181991577, 0.642098605632782, -0.06801434606313705, -0.11221513897180557, -0.30613213777542114, -0.8773781657218933, 1.4536207914352417, -0.2918783724308014, -0.41455167531967163, 1.216259241104126, -0.02158910036087036, 0.07182838767766953, 1.0076780319213867, -0.7759730815887451, -0.6775530576705933, 0.8699404001235962, -0.4021363854408264, -0.17033787071704865, -0.7064419388771057, 2.13275146484375, 1.8255223035812378, -1.0593459606170654, 0.45547839999198914, -0.21642647683620453, -0.9906754493713379, -0.1306656301021576, -0.800820529460907, 0.28585678339004517, 0.2859191298484802, 0.8185268640518188, 0.0490758940577507, 2.1158759593963623, 1.6968071460723877, 1.8985238075256348, 0.010127409361302853, -0.7794955372810364, -0.8186898827552795, 2.0400023460388184, -1.3508161306381226, 0.4080314040184021, 0.03421156853437424, -0.8043286800384521, 0.0747670829296112, 1.5160232782363892, 0.869463324546814, -1.0396002531051636, -0.40371131896972656, 1.3372100591659546, -0.3457934856414795, 0.6263850927352905, 1.1839255094528198, 0.038307949900627136, -0.20167428255081177, 0.20398183166980743, -1.115766167640686, 0.49383819103240967, 1.395468831062317, -1.9461045265197754, 2.5765910148620605, -0.24014973640441895, -0.9004629254341125, 0.4366162419319153, 0.04422471672296524, -0.12152771651744843, 1.0269304513931274, 0.2561244070529938, -0.9829571843147278, -1.2870314121246338, 0.44657662510871887, 0.4258951246738434, -1.7090966701507568, -1.0679106712341309, -0.9928019642829895, -0.2963126301765442, -0.318449467420578, 1.4290415048599243, -0.03199300915002823, -1.1791231632232666, 0.9000323414802551, 0.24630244076251984, -0.28175923228263855, -2.083613634109497, -0.03800005093216896, -1.9071491956710815, 0.8014316558837891, -1.2413030862808228, -0.5322567820549011, -0.14966586232185364, 0.3945961892604828, -1.1376562118530273, 0.9355284571647644, -1.1610863208770752, 0.012084339745342731, -0.45458242297172546, -1.5106738805770874, 0.14944985508918762, 1.557963490486145, 0.05899422988295555, -0.5258286595344543, 0.3242796063423157, 0.14056873321533203, -0.20747317373752594, -0.8380531072616577, -1.4502580165863037, 1.0756433010101318, -2.384476900100708, 0.5278165340423584, 0.438652902841568, 1.119884967803955, -0.22993986308574677, 0.22405236959457397, 0.5435740351676941, -0.8538905382156372, -0.14084307849407196, -0.688944935798645, -0.21434062719345093, 0.24925760924816132, 0.8028906583786011, 1.4538745880126953, 0.11283445358276367, 0.09338792413473129, 0.432371586561203, -0.5835219025611877, 1.2392815351486206, 0.6846833825111389, -1.5795626640319824, -0.9202082753181458, 0.5468029379844666, -0.47567200660705566, -0.5391581654548645, -0.5761483907699585, 1.6092705726623535, 1.1979619264602661, -1.606908917427063, -1.1188017129898071, 0.1401531845331192, -1.7945579290390015, 0.5038855075836182, 1.0296522378921509, -0.6316300630569458, -0.8378907442092896, 0.6988952159881592, -1.2702510356903076, 2.503932237625122, 0.32456454634666443, 0.234671488404274, -0.6544645428657532, 1.8205598592758179, -0.11096768826246262, 0.7759998440742493, 0.2891191244125366, -0.7181664109230042, -0.5596407055854797, 0.1052095964550972, -1.3900433778762817, 0.5222809910774231, -0.6897312998771667, -0.04910663142800331, -0.9991747140884399, -0.006217981688678265, -1.769903540611267, -1.7835170030593872, -0.748916506767273, 0.6916004419326782, 1.3147494792938232, -0.45521295070648193, -0.25456786155700684, 0.5278615951538086, -1.4419602155685425, -0.8133334517478943, -0.008487074635922909, -0.33194732666015625, 0.23204471170902252, -1.1207101345062256, -0.20115700364112854, 0.9108837246894836, -1.1820052862167358, 0.06355730444192886, 0.1868634670972824, 0.9985330700874329, 0.8445019125938416, 1.2803058624267578, 1.6429332494735718, 3.405303716659546, -1.7110322713851929, 1.1567277908325195, 0.5268872380256653, -1.1201045513153076, 1.1873594522476196, -1.254646897315979, -0.9036970734596252, -1.871256709098816, -1.6066937446594238, 1.5137217044830322, 0.10179511457681656, 0.3475266396999359, 0.5033748745918274, -1.271618127822876, -0.5003936290740967, 0.09319432824850082, 0.054982028901576996, 1.8269116878509521, -0.27956801652908325, -0.6144214272499084, -0.9689426422119141, -1.1927934885025024, -0.6313275694847107, -0.8317966461181641, 0.3898947238922119, -0.39121997356414795, -1.140873670578003, -1.2031835317611694, -0.24174438416957855, -0.7453365325927734, 0.37908560037612915, 0.8646543025970459, -0.9562906622886658, 0.8287451863288879, -0.5965794324874878, -0.3955351710319519, 0.5734788775444031, 1.0524930953979492, -0.31730547547340393, 1.7203187942504883, -2.0890989303588867, 0.06690102815628052, 1.4781148433685303, -0.5716235637664795, 0.5122388005256653, -1.7095575332641602, 1.088561773300171, -0.6571959257125854, 0.7982277870178223, 2.0577995777130127, 0.19464737176895142, -0.32361167669296265, 0.5603563189506531, 0.2714861035346985, 1.0290173292160034, 1.910738229751587, -0.08997954428195953, 2.349661111831665, -0.11920548230409622, -1.4017462730407715, 0.20398060977458954, -0.02734016813337803, -0.48220011591911316, 0.716255784034729, -0.18797940015792847, -0.4929267168045044, 0.2004004716873169, -0.40651950240135193, -0.09237947314977646, 0.14227840304374695, -0.7202656865119934, -0.12746375799179077, -0.4130651652812958, -0.15015430748462677, 0.3672732412815094, -1.2923767566680908, -2.1993448734283447, -0.06070692464709282, 0.24496866762638092, -0.22810505330562592, -0.25328588485717773, -0.9070162773132324, -1.5960850715637207, 0.08973094075918198, 0.8203177452087402, 0.12809628248214722, 0.764723539352417, -0.8503777384757996, 1.3450766801834106, 2.4880361557006836, -0.23070871829986572, -0.15031270682811737, -0.5832374691963196, 0.5601776838302612, 0.8116126656532288, -1.2162271738052368, 2.237157106399536, 0.590937614440918, 0.4985203444957733, -1.1771782636642456, 1.2256191968917847, -0.026174498721957207, -0.23298616707324982, 1.5426537990570068, 0.8871373534202576, -0.3720022737979889, 0.4259101450443268, -0.45164528489112854, 0.348984032869339, 0.30194947123527527, -0.27939072251319885, -0.09392989426851273, 0.7324493527412415, -0.9733847975730896, -0.7683209180831909, -0.12476533651351929, 0.4178542196750641, -0.6847026348114014, 2.036423444747925, 0.0666450783610344, 2.092726230621338, -1.0984166860580444, 0.654473602771759, 0.09883645176887512, 0.9443584084510803, -2.999506950378418, -0.8507953882217407, 1.704089641571045, -0.6237859725952148, 0.1424839049577713, 0.8811883330345154, 2.3324623107910156, -0.5583322644233704, -0.12539736926555634, 0.21287287771701813, -1.5264250040054321, -0.3968278765678406, -0.4857129454612732, 0.12501326203346252, 1.752380132675171, 1.649229645729065, -0.1531098634004593, -0.7762449979782104], [0.5312752723693848, 1.6591674089431763, -0.5195257663726807, -0.8778945207595825, 1.1954903602600098, 1.310590386390686, 0.9384272694587708, 2.099360942840576, -0.16894960403442383, -0.47527575492858887, 0.6582354307174683, 0.8320094347000122, -0.7213571667671204, 1.0252633094787598, -2.194885492324829, 0.3973413109779358, 0.7489414811134338, -1.861934781074524, 1.3176850080490112, 1.2850191593170166, 0.27440527081489563, -0.213458850979805, 0.5985321402549744, 0.5844821333885193, 0.5914268493652344, -0.8100637793540955, 0.9063379764556885, -1.3708833456039429, 1.759494662284851, 1.503660798072815, 0.19814714789390564, 1.5545710325241089, -2.5656354427337646, 0.6663287878036499, 0.5288040637969971, 0.690639317035675, 1.280657172203064, 0.2619255781173706, -0.8946475982666016, -1.3401330709457397, 0.5708303451538086, 0.1476958841085434, 1.0363194942474365, -0.885251522064209, 0.34965333342552185, 0.025859083980321884, 1.9344335794448853, -0.5309504866600037, -1.6646753549575806, -0.8853256702423096, -0.6318628787994385, -1.5664722919464111, 0.9518669247627258, 0.07425738871097565, 0.07470640540122986, -0.33626773953437805, 0.3714919090270996, -1.8586947917938232, 1.3426703214645386, 0.7857221364974976, 0.9243856072425842, -1.1715470552444458, 0.15134096145629883, -2.008882761001587, -1.9925429821014404, 1.8912769556045532, -0.30756670236587524, -1.6722570657730103, 1.0709437131881714, -0.7068649530410767, -0.3791760206222534, 0.732623279094696, -0.47651344537734985, 1.214637041091919, 0.2319381684064865, 0.21733877062797546, -0.15572944283485413, -0.4446440041065216, -0.6844009160995483, -0.8336769938468933, 0.7157617807388306, 0.3046664893627167, -1.690531849861145, -0.5141929388046265, -0.6279045939445496, 1.6628501415252686, -0.7432134747505188, -0.3783602714538574, -0.32022884488105774, -0.9244564175605774, 0.41979461908340454, -1.1073209047317505, -0.3022196888923645, 1.3955481052398682, -0.5169890522956848, -0.12535834312438965, -1.7529770135879517, -0.7803651690483093, 0.8267956972122192, 0.47365620732307434, -0.4910582900047302, -0.2308797389268875, -1.0724270343780518, 0.317382276058197, 0.10350670665502548, -0.5603325366973877, -1.6891957521438599, -1.2809463739395142, -0.4188365042209625, -0.5810232162475586, -0.1893901824951172, -1.1182833909988403, -1.819093108177185, -1.0699598789215088, -2.9682440757751465, -0.6658071875572205, 0.807636022567749, -0.23341436684131622, -1.0948814153671265, 2.2664573192596436, -0.7005868554115295, 0.3730693757534027, 0.6684612035751343, -0.65716552734375, 1.0472699403762817, 0.8589797019958496, 2.2144484519958496, 0.29902684688568115, 0.48381537199020386, 0.3091640770435333, -0.5837397575378418, -0.3360377848148346, 2.197690725326538, 0.6600980758666992, -1.497046709060669, -1.3198035955429077, -0.3269340395927429, 0.215164452791214, -1.4944961071014404, -0.48918065428733826, -1.3160165548324585, -0.21793095767498016, -0.46274513006210327, 0.7080005407333374, -2.337639093399048, -0.2680083215236664, 2.195784330368042, 0.4769325256347656, 0.5492335557937622, -0.8568636775016785, -1.967490792274475, -1.493852972984314, 1.1616756916046143, -0.31288355588912964, 0.7197615504264832, -1.7119617462158203, 0.3697583079338074, 1.0203077793121338, 0.13527719676494598, -0.1958169788122177, -1.3259203433990479, 0.0934356227517128, 0.22838103771209717, 1.0153026580810547, 1.2289494276046753, -0.8840365409851074, 0.3534452021121979, -0.707243800163269, 0.4768940806388855, -1.1044549942016602, -0.24059079587459564, -0.33902889490127563, 1.272325038909912, -0.004284997936338186, 1.2635754346847534, -0.274840384721756, 1.0986310243606567, 0.6121232509613037, -0.11593925952911377, -0.03016742691397667, -0.05201395973563194, 0.46859362721443176, 2.044081926345825, 0.32265400886535645, -1.1893590688705444, -0.1566428691148758, -0.025176698341965675, 0.8594664335250854, -1.6275966167449951, -0.3769434094429016, 0.4121229350566864, 0.9271199703216553, -0.3671708106994629, 0.10634112358093262, 1.9791978597640991, -0.9588684439659119, 1.0528357028961182, 0.5206893682479858, 0.44358953833580017, 0.41740313172340393, 0.9117162227630615, -0.22179196774959564, -0.18829596042633057, -0.9166949391365051, 1.1066263914108276, -0.20613406598567963, -1.2200499773025513, -0.48113560676574707, -0.3694378733634949, -1.8230689764022827, 1.121052622795105, 0.34646525979042053, 0.3315886855125427, -1.6497750282287598, 0.3361803889274597, -1.102662444114685, 0.16974912583827972, -0.9029931426048279, 0.5841989517211914, 0.7804110646247864, 0.0784507468342781, 1.6398471593856812, -0.4641357958316803, -2.067216157913208, -0.4885174632072449, -0.8705620765686035, -1.1456528902053833, 1.1905220746994019, -0.8703552484512329, -0.23480311036109924, -0.08241309970617294, 0.6856384873390198, -0.46188676357269287, -0.12125267833471298, 0.7053117156028748, -1.2074731588363647, 1.1435497999191284, -1.187959909439087, 0.2710602581501007, 0.36340367794036865, 0.6412844657897949, 1.0877004861831665, 2.1858787536621094, 0.7160791158676147, 0.7348071932792664, -0.6549115180969238, 0.2698763310909271, -0.7498493194580078, -0.48734086751937866, -0.5030022263526917, -0.17363417148590088, -0.9463402628898621, 0.22482262551784515, -0.488919734954834, 1.5020555257797241, -0.7268403768539429, -0.44346675276756287, 1.2103067636489868, -0.6876415610313416, 0.5403741002082825, -0.8014107942581177, -0.05563432350754738, -1.9085593223571777, -1.1740361452102661, -1.941710114479065, 0.013981903903186321, -0.401868999004364, -0.5601974129676819, -0.19805105030536652, 1.1618176698684692, 0.9084813594818115, 0.30781516432762146, -0.6624118089675903, 0.9735419154167175, -0.9068780541419983, -0.8198967576026917, 1.7862900495529175, -0.16515225172042847, 0.3431363105773926, -2.6145265102386475, 0.12453503906726837, 1.6674209833145142, -1.8860669136047363, 0.19521430134773254, -0.5935702323913574, -0.25518321990966797, 1.3888133764266968, 0.20762421190738678, 0.46552637219429016, 0.100124292075634, -2.0184991359710693, -0.9948804974555969, 2.2095789909362793, 0.7047573328018188, -1.0214308500289917, 1.7670649290084839, 1.0554169416427612, 0.2514609694480896, 0.1682947725057602, 0.6557000875473022, -0.32598036527633667, 0.9388819336891174, 0.4517594575881958, -0.1320050060749054, 0.5592151284217834, -0.6490228176116943, 1.1572248935699463, 0.027782369405031204, -0.9347316026687622, 1.4360324144363403, -0.5216611623764038, -0.800349771976471, -1.0953444242477417, -0.3809320330619812, -0.3454880714416504, 0.0500040277838707, 0.22025716304779053, 0.975481390953064, -0.5952023267745972, 0.03678641468286514, 0.09224629402160645, -1.9035547971725464, 0.2504534125328064, -0.9812834858894348, 0.05324632301926613, 0.07028897106647491, -0.6648560762405396, -1.9640206098556519, -1.004088282585144, 2.2529664039611816, -1.0412826538085938, 1.0263487100601196, -0.12200822681188583, 1.0846275091171265, -0.8698747754096985, 0.6236182451248169, 0.008058667182922363, -0.0274124126881361, -1.7003897428512573, 0.6741202473640442, 0.363444447517395, 0.5716289281845093, -0.6982159614562988, 0.21182478964328766, -0.36020609736442566, 0.4059554934501648, -1.2929909229278564, -1.430985689163208, -1.520788550376892, 0.7419686317443848, 1.2992891073226929, -0.5656936168670654, 1.4472209215164185, 0.9937295317649841, -0.6982289552688599, 0.6500522494316101, 1.3204302787780762, 1.3200348615646362, -1.0922601222991943, -0.8272475600242615, 0.6367392539978027, 1.377931833267212, 1.969064712524414, -0.8113541603088379, -1.1744250059127808, 1.2132718563079834, -1.6308205127716064, -0.4533792734146118, 0.23803338408470154, -1.6923165321350098, -1.120218276977539, -0.20526348054409027, 0.19757990539073944, -0.43597424030303955, 0.5902867913246155, -0.8061034083366394, -0.4013875424861908, 0.6059243083000183, 0.5887131094932556, -1.311471939086914, 2.096283435821533, -0.5726165771484375, 1.5638097524642944, 0.11530619114637375, -1.095418930053711, -1.630066990852356, -0.5930080413818359, 1.9413484334945679, -0.5616907477378845, 0.4448397755622864, 0.3906879127025604, 0.2823476493358612, -0.5388748049736023, 1.5227792263031006, 0.2906413972377777, 1.5566688776016235, 0.2548213005065918, 1.0570813417434692, 1.0468721389770508, 0.8508619070053101, -1.5719467401504517, 1.0342849493026733, 1.6548004150390625, -1.444663643836975, 1.6021363735198975, -1.6496329307556152, -0.6630157232284546, -0.9828504920005798, 0.6088881492614746, 0.22171080112457275, 0.7498743534088135, 0.10300558060407639, -0.5029668807983398, 0.5940257906913757, -0.9641515612602234, 0.3123440146446228, 0.1554117053747177, -0.4119802415370941, -0.36435607075691223, -1.234192132949829, 0.4686048626899719, 0.40290704369544983, -0.28997907042503357, 0.3047669231891632, -0.6487746238708496, 0.2551482617855072, 1.567063570022583, -2.0220470428466797, 0.28773438930511475, -0.5182284712791443, -0.26879188418388367, 0.5521560907363892, -1.5506837368011475, -0.20552141964435577, -0.7793723940849304, -0.9140578508377075, -0.8386213779449463, -1.808555245399475, -0.24995997548103333, -0.29227378964424133, 1.4473390579223633, -0.9740722179412842, 0.3027612864971161, -0.5681636333465576, 0.7508199214935303, -0.32506370544433594, -0.2871512174606323, -0.6226370334625244, -0.8161059021949768, 1.3605681657791138, -2.4599218368530273, 0.4082055985927582, 0.595725953578949, -1.0098904371261597, -0.4040449559688568, 0.9460636377334595, 1.3564786911010742, -0.480754554271698, 1.3129445314407349, 0.3145076334476471, -1.6477948427200317, -1.0008254051208496, 0.21199457347393036, -0.8701605796813965, -0.38984671235084534, 0.024559833109378815, 0.5297690033912659, 2.0037758350372314, -0.5667341351509094, -1.0497028827667236, 0.20830486714839935, -0.18473181128501892, -0.7807669639587402, 0.0815635398030281, -1.948230266571045, 1.477319598197937, 1.0315104722976685, -0.42148903012275696, -0.5189439058303833, -0.18366460502147675, -0.5718197226524353, -0.7563415765762329, 1.1798549890518188, -1.100735068321228, 0.42272287607192993, -0.16214516758918762, 1.721588373184204, 1.2131627798080444, 0.2549281418323517, 0.040279947221279144, -1.3116397857666016, -0.12956340610980988, 0.5766858458518982, -1.4505250453948975, 0.11965876817703247, -1.463382363319397, -0.9490877985954285, 1.357965350151062, 1.1076902151107788, -1.2136738300323486, -1.028352975845337, 1.4955142736434937, -0.9011039733886719, 0.7847530245780945, -1.5163120031356812, 0.2321462780237198, -0.31234756112098694, -0.3914147615432739, -1.2773939371109009, -0.40380698442459106, 0.506599485874176, -0.4149025082588196, -0.7957959771156311, 0.016737166792154312, 1.1219431161880493, 1.1544427871704102, 1.3991419076919556, -0.18751955032348633, -1.1187587976455688, -0.7577018737792969, 0.9194114208221436, -0.8269535303115845, 1.8674484491348267, 0.8342657089233398, -0.5834754705429077, -0.33211207389831543, -0.27791866660118103, 2.0554263591766357, -2.795780658721924, -1.5324918031692505, 0.9660340547561646, -0.7094156742095947, -0.3869856297969818, -0.34588226675987244, 1.8558021783828735, -0.30031123757362366, -2.4041028022766113, 0.04826074838638306, -0.6801530122756958, -0.220456063747406, -0.2563649117946625, 0.9649374485015869, 1.4301930665969849, 1.0306402444839478, 1.0237709283828735, 1.6655176877975464, 0.6515273451805115, -0.7701049447059631, 0.9760303497314453, 0.3261389434337616, 1.684112787246704, 0.4299033284187317, 1.026389479637146, -0.9195461273193359, 0.033952876925468445, -0.04878930002450943, 0.9891034960746765, -0.07774807512760162, 0.6859171390533447, -0.9769221544265747, -0.3317824900150299, -0.06824656575918198, 0.7703296542167664, -1.7577496767044067, 1.0193893909454346, -0.11085127294063568, -3.0447895526885986, 0.7860326170921326, -0.8166168332099915, 1.4717164039611816, 2.1020052433013916, 0.31502199172973633, -0.30483996868133545, -0.6819478273391724, 0.9753008484840393, -0.7162512540817261, -0.38103359937667847, -1.5600157976150513, 0.4218350946903229, -1.6964772939682007, -0.3400215804576874, -0.24257595837116241, 0.4606928825378418, 0.4684973955154419, -0.7320452928543091, 0.4429984986782074, 0.07853472977876663, -0.5588974356651306, 0.3067416548728943, -1.0243124961853027, -1.2356008291244507, -0.6012117862701416, -0.8419360518455505, -1.081093430519104, 0.8015642762184143, -0.7467857599258423, 0.6978377103805542, -0.471113920211792, -1.000968098640442, 0.7296140193939209, 1.099386215209961, 0.5302827954292297, -0.2677222788333893, -1.509644865989685, -0.3934638798236847, 0.7876391410827637, -1.6234674453735352, -3.010364294052124, -1.3693721294403076, -0.4073706269264221, -1.0846149921417236, 0.18549473583698273, 0.9527456164360046, -0.38195616006851196, -0.13767366111278534, -0.08099234849214554, 0.5413526892662048, 0.8597134351730347, 1.2270351648330688, 0.585169792175293, -0.582355260848999, 0.12250224500894547, -0.5054716467857361, 0.8572505116462708, -2.3068907260894775, 0.045555148273706436, -1.2601734399795532, -0.8038526773452759, 0.44181832671165466, 0.5390911102294922, 0.4510105848312378, 1.4927231073379517, 1.2250208854675293, -0.9660512804985046, -0.5660885572433472, 0.026525935158133507, 0.22650162875652313, 1.5348472595214844, 0.8821831345558167, -0.8641007542610168, 1.071602463722229, 0.2121741771697998, -0.013697235845029354, -1.1378690004348755, -1.1835839748382568, 0.9212059378623962, 0.17129723727703094, 1.2282434701919556, 0.37214258313179016, 0.2453378140926361, 0.041308894753456116, 0.32109326124191284, -0.5182846188545227, -0.05159936472773552, 0.18564006686210632, -1.743804693222046, -0.4439171552658081, -0.77947598695755, 0.5089854001998901, 1.3371366262435913, -0.35303065180778503, 0.6795167922973633, -1.30086088180542, -0.6188860535621643, 0.4172877073287964, -0.019691672176122665, -2.242832660675049, -0.282767653465271, -0.3757416009902954, 1.5724364519119263, 2.131714105606079, -3.4585776329040527, -1.2575911283493042, 0.7815226316452026, -0.6492013335227966, -1.0450469255447388, -0.22688551247119904, -0.5790548324584961, 0.3213343620300293, -0.8874518871307373, 0.8411651849746704, 0.4027147889137268, -0.44809654355049133, 0.5361310243606567, 0.6132180094718933, -1.7305272817611694, -0.49167758226394653, 0.5361326336860657, -1.4685463905334473, 0.6642612814903259, -0.27386537194252014, 0.561120867729187, 2.15043306350708, -0.8988370299339294, 0.12549051642417908, -0.0028597309719771147, 0.38439202308654785, -0.5276534557342529, 0.22078190743923187, -0.06614784896373749, -0.7011014223098755, 0.9092273712158203, -1.647939920425415, 0.3604230582714081, 0.1491110622882843, -1.3833069801330566, 0.0058711739256978035, -3.135430335998535, 0.007046212907880545, 0.9741501212120056, 0.1377106010913849, 0.4775298833847046, -0.18880105018615723, 1.9557485580444336, -1.4483156204223633, 0.21025308966636658, -0.6302441358566284, -0.009710807353258133, 0.6368236541748047, -1.003572702407837, -1.1214977502822876, -0.3611590266227722, 0.09735944122076035, -2.5091123580932617, 1.3491315841674805, 0.17646948993206024, -0.856568455696106, -0.22472406923770905, -1.4068856239318848, 1.2754302024841309, 0.6067955493927002, -0.9412782788276672, -1.7813482284545898, 0.3071501553058624, -0.497444212436676, 0.35709288716316223, 1.1731711626052856, 0.7561697959899902, -2.2738399505615234, 0.01197375450283289, 1.7914048433303833, -1.0692460536956787, -0.6856157183647156, 1.2684844732284546, -0.7504116296768188, -0.048787008970975876, 0.874568521976471, 0.5118349194526672, 0.2470596730709076, -0.13660521805286407, -0.22779855132102966, 0.8518056273460388, 0.6389352083206177, -0.3691159188747406, 0.42505142092704773, -0.08815581351518631, -0.14357411861419678, 0.2593200206756592, 0.13795332610607147, -1.6706277132034302, -2.1486170291900635, 1.188962697982788, 0.09648267924785614, -0.1672120839357376, 0.743295431137085, 1.0601335763931274, -0.9729439616203308, -0.48462027311325073, -0.6545234322547913, -1.182112455368042, 0.8489335775375366, -0.10531666874885559, 0.3700152039527893, 0.5221086740493774, -0.2761205732822418, 1.0694489479064941, 1.0887621641159058, 0.09385716170072556, 0.3961428999900818, -0.05083194002509117, 0.06277336925268173, 0.1585947722196579, -0.5753196477890015, 1.207767367362976, -1.1659317016601562, 0.4496791958808899, 1.900596261024475, -1.3674484491348267, -0.03801451250910759, 0.5447686910629272, -0.8514190316200256, 0.02843344397842884, -0.9224952459335327, -0.18543240427970886, -1.3795478343963623, 1.1912678480148315, 1.2928870916366577, -0.00136575975921005, -1.7463077306747437, 0.4196956157684326, 0.46036380529403687, -1.2560323476791382, -0.9196177124977112, 0.5608704090118408, 1.0702930688858032, -0.8090700507164001, 0.4771040081977844, 1.8478152751922607, -1.3791513442993164, -0.9332589507102966, -1.1804956197738647, 0.13415655493736267, -0.8375223278999329, 0.6450154781341553, -1.5591744184494019, -0.6958550810813904, -0.2189057469367981, -0.6366201043128967, -1.542023777961731, 1.2666572332382202, -0.6609749794006348, -0.8186603784561157, -1.80343759059906, 0.5142561793327332, 0.7657366394996643, -0.01975322887301445, -0.21516935527324677, -1.529727816581726, -1.0816770792007446, -0.44677308201789856, 0.5984291434288025, -0.07047832757234573, -0.22073359787464142, 0.45810672640800476, 0.9010764360427856, 1.4345873594284058, 0.5084935426712036, -0.4483173191547394, 0.5074876546859741, 0.8681607842445374, -0.9301890134811401, -0.2484906166791916, 2.234375238418579, 2.184565305709839, 0.5057932734489441, -0.4123937785625458, 0.40708479285240173, 0.5901316404342651, -0.060596246272325516, -0.07416938245296478, -0.4147847890853882, -0.7313308119773865, -0.9069517254829407, -0.4221491515636444, -0.782326340675354, -1.4874634742736816, 0.5278586745262146, -0.09809096902608871, 1.819502592086792, -0.773224949836731, 0.5293527245521545, 0.581737220287323, 0.28993454575538635, 1.2932112216949463, -1.4152880907058716, -0.07548883557319641, -0.8306808471679688, 0.11299218982458115, 0.10706431418657303, 0.157209575176239, -0.8547773957252502, 0.8168294429779053, 0.799977719783783, -0.14234763383865356, 0.6741513609886169, 1.1666173934936523, -0.6660174131393433, 1.3144478797912598, 0.04720839485526085, 0.8462415933609009, -1.5444282293319702, 0.7884528040885925, 0.6746900081634521, -0.836311399936676, 1.2915267944335938, -0.49616754055023193, 0.4860111474990845, -0.1463685929775238, -0.5705170631408691, -0.027959411963820457, 0.26601454615592957, -0.32725989818573, 1.0627537965774536, -1.2305874824523926, 0.11844469606876373, 0.3890112340450287, -1.0594565868377686, 1.3924837112426758, -2.059704542160034, 0.48966044187545776, 1.2661914825439453, 2.8952314853668213, 1.1107255220413208, -1.8615564107894897, -0.004165025893598795, 0.3384978175163269, -0.18804261088371277, 1.0722326040267944, -1.716792345046997, -1.547706961631775, 0.37860360741615295, 0.5176995396614075, 0.8837685585021973, 0.40438899397850037, -0.13458512723445892, -0.8808053731918335, -0.9534722566604614, 1.3976465463638306, 1.1073927879333496, 0.8635351657867432, -0.14090238511562347, 1.2138526439666748, 0.4834122657775879, -0.16493067145347595, 0.341194212436676, -2.573960065841675, -0.9415878653526306, -0.21361050009727478, 0.3285503387451172, -0.23177872598171234, -1.0969434976577759, 0.812777578830719, 1.2488267421722412, 0.0736398696899414, 1.7246532440185547, -0.17279919981956482, -0.5646412372589111, -0.03049515187740326, -0.15331877768039703, 2.1581437587738037, 1.7900609970092773, -0.6521338820457458, 0.4759838581085205, -1.332789421081543, 0.8834125995635986, 0.669013500213623, -0.15627911686897278, 0.7827135920524597, 0.3817436695098877, 0.9994505643844604, -0.5355928540229797, 0.6999657154083252, 0.7858275175094604, -1.66254460811615, 1.483963966369629, -0.7813084125518799, -0.07480692118406296, -2.1780028343200684, -0.496383935213089, 0.8912547826766968, 1.240716814994812, -0.11307484656572342, 1.1209192276000977, -0.14254437386989594, -0.9644230604171753, 0.466329425573349, 0.2888270914554596, 0.40217921137809753, -1.3828010559082031, 0.1653684377670288, 0.15249036252498627, 1.6012932062149048, 0.014461585320532322, -0.07342445850372314, 0.9765841364860535, 1.4181180000305176, 0.9308189749717712, 1.0821064710617065, 0.9841504096984863, -0.2134489268064499, -1.2638709545135498, -0.000102797886938788, -0.6020558476448059, -0.6911662220954895, -0.36921533942222595, -0.5316734313964844, -0.7268494367599487, -0.30842679738998413, -0.26645627617836, -1.2160046100616455, -0.903369665145874, -0.5305222868919373, -0.4912753999233246, -0.47329476475715637, -0.7158094048500061, 0.8391144871711731, 2.0621252059936523, 0.38777226209640503, 0.46286842226982117, -0.8474038243293762], [-0.6271855235099792, 1.3426264524459839, -0.447915256023407, 1.0474717617034912, 0.054177768528461456, 0.27439063787460327, 0.007678018417209387, -0.6751158237457275, 0.5639544725418091, -0.6087969541549683, 1.3369420766830444, 0.12511660158634186, -0.6312776803970337, 0.25975120067596436, -0.2898254692554474, -0.5801841616630554, 0.5456026196479797, 1.1416765451431274, -1.2403432130813599, -0.3945172131061554, 1.6102322340011597, 0.368282675743103, 0.28753799200057983, -1.21392822265625, -0.07800678163766861, -0.3247714042663574, -0.7964730262756348, 0.06926785409450531, 0.03890419378876686, -1.3177944421768188, -0.32381635904312134, 0.4970547556877136, -0.44389739632606506, -1.4046213626861572, 1.302522897720337, -0.10936732590198517, 0.05862267687916756, -1.1965540647506714, 0.15232156217098236, -0.9658739566802979, -0.24613207578659058, 1.1673020124435425, 0.9292761087417603, 0.5435423254966736, -0.962215006351471, 0.9584687352180481, -0.6645184755325317, 0.20324063301086426, 0.6162398457527161, 0.3220140039920807, 0.6228219270706177, -0.8335665464401245, -0.3088656961917877, 0.7972568273544312, 0.7461130023002625, -0.274189829826355, 0.6988563537597656, -1.6834510564804077, -0.8302759528160095, 0.13415351510047913, 0.5053418874740601, -0.06599289178848267, 1.4039140939712524, -2.0978000164031982, 1.0653631687164307, -0.9111176133155823, -0.014561320655047894, -0.30641257762908936, -1.1007764339447021, -1.0959975719451904, 1.7422915697097778, -2.084108591079712, -1.3710166215896606, 1.476579189300537, -0.2666740417480469, -0.8101701736450195, 0.24097296595573425, 0.42078080773353577, 1.4267429113388062, -0.4369831383228302, -0.1698029637336731, 0.9675244092941284, 0.15685045719146729, 0.9513485431671143, -1.639853835105896, -1.0968585014343262, 0.13319522142410278, -1.4112070798873901, 1.5770468711853027, -0.8744257092475891, 0.393404096364975, -0.6566958427429199, -0.05535572022199631, -0.00831593107432127, 0.49599596858024597, 0.8179537653923035, -0.4613282382488251, 0.9274072647094727, 0.26177823543548584, 1.9273622035980225, 0.4153956472873688, -1.6770426034927368, 0.748552143573761, -0.3441888391971588, 0.08819817006587982, 0.511333167552948, -0.3371727764606476, 1.305959939956665, -1.270375370979309, -0.4628431499004364, 0.6246037483215332, -0.05041103437542915, -0.7420594692230225, -0.0253619235008955, -0.28614431619644165, -0.3271730840206146, -0.05519193783402443, -0.03347672522068024, -0.6567130088806152, -2.514286518096924, -0.5846889615058899, 0.6553257703781128, -0.24109072983264923, -2.3960886001586914, -0.5118787288665771, -0.14936508238315582, -0.1549435704946518, 0.691512942314148, 1.426074743270874, -0.50022292137146, -0.14913132786750793, -1.5275112390518188, -1.8551249504089355, 0.3326970934867859, -0.9374268054962158, -0.8147692084312439, -0.37246114015579224, -0.11163213849067688, -0.020743969827890396, -1.0353556871414185, -0.7116543650627136, 0.9261255264282227, 0.7659499049186707, -0.7757275104522705, 0.6917950510978699, -0.20220713317394257, 1.440359354019165, 0.4456760585308075, 0.4219350814819336, 0.8053045868873596, -0.025565749034285545, 0.18392542004585266, 0.17923228442668915, 0.8640058040618896, -1.0803345441818237, 1.6042810678482056, -0.19305945932865143, -1.1650564670562744, -0.2801208794116974, 0.1599956601858139, -0.7864968180656433, -0.13358986377716064, 1.0795680284500122, 1.5412405729293823, -0.013161447830498219, 0.38145995140075684, 1.449789047241211, 1.7803303003311157, 1.4078004360198975, -0.7117423415184021, 0.864936888217926, -0.21163585782051086, 0.13619893789291382, -0.8762661814689636, 1.1446254253387451, 0.5836678743362427, -1.062080979347229, 0.31873729825019836, -1.1323410272598267, -0.8638111352920532, -0.47319501638412476, 0.4160996377468109, -0.7513434886932373, 0.4429531991481781, -0.10448814928531647, 0.4897952973842621, 0.8109467625617981, 0.2079884111881256, -0.09356603771448135, 0.611203670501709, 0.5687808990478516, 0.7773984670639038, -0.5433752536773682, -0.3202870786190033, -1.538694977760315, -0.4214645326137543, -0.011314568109810352, 0.02201228030025959, 0.858564555644989, -0.5958220362663269, 0.05771017447113991, 0.5189977288246155, 0.2306043654680252, 0.6382242441177368, 0.7859182357788086, 1.1952431201934814, -0.660099446773529, -0.3086927831172943, -0.6928316950798035, -0.6966468095779419, -0.24432769417762756, -1.56099534034729, -0.5980384945869446, 0.6422666311264038, -1.4451963901519775, 0.37846869230270386, 1.0995053052902222, 3.071611166000366, 0.6997044682502747, 0.31445735692977905, 1.1270432472229004, -0.29238393902778625, 0.764668881893158, 0.31621620059013367, 0.6207066774368286, 0.42396214604377747, -2.1084132194519043, -1.975271224975586, -0.8768113851547241, 0.6265775561332703, -1.9587624073028564, -1.39266037940979, -0.20568320155143738, -0.1664123386144638, -0.680348813533783, 0.26824355125427246, 1.1501117944717407, -0.8495049476623535, -1.056780219078064, -2.175104856491089, -0.936796247959137, -1.0771538019180298, -1.1564449071884155, -1.7564449310302734, -0.847284734249115, -0.4508466124534607, 1.0386831760406494, -0.47023746371269226, -0.6514528393745422, 0.8654021620750427, -0.5172958970069885, -0.273419588804245, 0.3413431644439697, 0.09200538694858551, -1.8023486137390137, -0.03693443164229393, 0.7406934499740601, -1.360094666481018, 0.8649935126304626, -0.6704156994819641, -0.7922489643096924, -1.9774699211120605, 0.12068530172109604, 0.23923633992671967, 1.63359797000885, -0.8326672315597534, 0.1033831238746643, 0.7600420713424683, -0.3807853162288666, -0.38092121481895447, 0.7875370383262634, 1.928481101989746, -1.5113320350646973, -0.5858249664306641, -0.6137811541557312, 2.393562078475952, 0.11702807992696762, 0.5263461470603943, -1.1496487855911255, 1.3856921195983887, -0.022421598434448242, 0.3469158709049225, 0.3143947124481201, -1.3442790508270264, -1.6737381219863892, 0.39630240201950073, 2.006495237350464, 1.000561237335205, 0.01983688399195671, -0.9339159727096558, -1.438434362411499, 0.12320872396230698, 0.9027498364448547, -0.8989530205726624, -0.2487732172012329, -0.07474355399608612, 0.34407925605773926, -0.5249194502830505, -2.592921733856201, -0.5825926661491394, -0.7917681932449341, -0.6089293360710144, -0.23908743262290955, 1.3042610883712769, 0.6830242276191711, 0.5367076992988586, -0.4667334258556366, 1.0124149322509766, -1.2603120803833008, 1.0523234605789185, 0.7258375883102417, 0.6377598643302917, 0.4882049560546875, -0.24228093028068542, -1.6323022842407227, -0.685036838054657, -0.4276110827922821, 0.05126947537064552, -3.6084177494049072, 0.43774181604385376, -0.8868677616119385, -0.7787778973579407, -0.1767183095216751, 0.26246708631515503, 0.41549405455589294, -0.0416744127869606, 0.8241162300109863, -0.3066504895687103, -0.6759112477302551, 0.3676096200942993, 0.2639060914516449, -0.3398543894290924, 1.2063133716583252, -0.18257050216197968, -1.5293927192687988, -0.5935338735580444, -0.17705535888671875, 1.1380671262741089, 0.044655121862888336, 1.5023226737976074, -0.6413580179214478, -0.9340515732765198, 0.21444310247898102, 0.1560138612985611, 0.23673412203788757, 1.4190773963928223, 0.7003107070922852, -0.27507641911506653, 0.14577706158161163, -0.823479413986206, -0.3130197525024414, 1.3306357860565186, 0.05316660925745964, -0.1714491844177246, 0.8850457072257996, -0.7000218629837036, 0.3657872676849365, -0.8786322474479675, -0.48258307576179504, -0.885300874710083, 0.15591642260551453, -0.42221248149871826, -1.0914344787597656, 0.2696686089038849, -0.08055590093135834, 1.3912200927734375, 0.3442051112651825, -0.5359601378440857, 1.3607791662216187, -0.7761748433113098, -0.027745164930820465, 0.6483660340309143, 0.12045587599277496, 0.5808935761451721, 0.9516317844390869, 1.4082467555999756, 0.7814961075782776, -0.006050082854926586, 0.2126084715127945, -0.3208593726158142, 0.09569092094898224, 0.34866446256637573, -1.3174329996109009, 0.8646571040153503, 1.5594182014465332, 0.15763571858406067, 0.6295494437217712, -1.890397548675537, -1.8111720085144043, 1.7416324615478516, -0.5550214052200317, 2.4675772190093994, -1.653546690940857, -0.7833368182182312, -0.526477038860321, 1.011724829673767, 0.7815331816673279, 1.4499539136886597, 2.0375399589538574, -0.33776143193244934, -0.10673090815544128, 0.7279458045959473, 0.5164808034896851, -2.309231758117676, 0.7337759733200073, 0.29661229252815247, 0.9557740688323975, -0.6164835095405579, 0.11380364000797272, 0.6782709956169128, 0.7659735679626465, 0.9374932050704956, 0.4147920310497284, -0.14390678703784943, 0.42554521560668945, -0.6733494400978088, -1.3809117078781128, 1.4602220058441162, 0.22113066911697388, -0.17340464890003204, 1.002007007598877, -2.2380216121673584, -0.461833119392395, 0.843438982963562, -1.1689956188201904, -0.19299426674842834, 0.036139920353889465, -0.445611834526062, 0.38311994075775146, -0.3263680338859558, 1.1463189125061035, -0.4156348705291748, 0.3856383264064789, -0.5088514685630798, -1.3213953971862793, -1.6337449550628662, 0.05821079760789871, 0.18962158262729645, -0.1834496706724167, 0.1393369734287262, 0.40434640645980835, -0.050487615168094635, 0.6895710825920105, -0.7923316359519958, 1.00875723361969, 0.6412773728370667, 0.5129771828651428, -0.42160695791244507, 0.9734413623809814, -1.6512795686721802, -0.4790085256099701, -1.2892701625823975, 0.30455976724624634, 0.6688705682754517, 0.2335144430398941, 1.169468879699707, -0.6445558667182922, 0.520584762096405, 1.570959210395813, -1.3829594850540161, -0.5079212188720703, 1.0593541860580444, 1.3456883430480957, 1.6150870323181152, -1.4853239059448242, -0.7701627016067505, 0.34535953402519226, 1.9578697681427002, 1.675215482711792, -0.14149810373783112, 0.5372909307479858, -1.0282331705093384, 0.567091166973114, 0.5498799681663513, -0.5023356080055237, 0.1999998688697815, 0.19547845423221588, -2.272923231124878, 1.9048219919204712, -0.52579265832901, 1.114367961883545, -0.2935672998428345, 0.5569836497306824, 0.4476083517074585, 0.8014020323753357, 0.9093532562255859, -0.26373961567878723, -0.2663421332836151, -0.9302496910095215, 0.39553529024124146, -0.6840366125106812, 0.9838014245033264, -1.2099509239196777, -0.5775054693222046, 0.47067299485206604, 1.148979902267456, 0.18038281798362732, -1.5532366037368774, -0.30082452297210693, -0.619792640209198, -0.8821921944618225, 0.3924790024757385, -0.3553507328033447, 0.7976049184799194, -1.9106017351150513, 1.975823163986206, 0.7529715895652771, 0.3451124131679535, 0.6952996850013733, -0.42027854919433594, 1.5594134330749512, 0.6763558387756348, -2.2455461025238037, 0.18061456084251404, 0.39645326137542725, 0.3734230399131775, 0.8912795186042786, -1.454939842224121, -0.20781178772449493, -0.2203030288219452, 0.0923352986574173, 0.08686453104019165, -0.6850491166114807, -0.013092181645333767, -2.251725673675537, -0.5304121971130371, 0.16790513694286346, 0.9979333877563477, -0.08500059694051743, 1.6477375030517578, 0.8191831707954407, -0.6186138391494751, 0.37520337104797363, -1.2089171409606934, -0.9164575338363647, 0.21543709933757782, 1.2855958938598633, 0.5951708555221558, -1.3070971965789795, -0.06683340668678284, 1.0673134326934814, -0.8341367840766907, -0.28387120366096497, 0.6292434930801392, -0.09448767453432083, 0.3593195974826813, -0.5726274251937866, 2.3339731693267822, -0.42173323035240173, 0.4309736490249634, 0.1808786690235138, -0.37215399742126465, 0.7917206287384033, 0.8690701127052307, 0.3658207654953003, 0.017142336815595627, -2.6947829723358154, -0.7413122057914734, 0.2836553156375885, 0.04904910922050476, 0.6438093185424805, -0.8270589113235474, -1.3438481092453003, 0.9225441217422485, -0.6304686069488525, 0.6484363079071045, 1.5543639659881592, -0.18270038068294525, -0.041272345930337906, 0.9311956167221069, 0.43373844027519226, 0.4199848175048828, -1.6214323043823242, 0.4605039656162262, -1.12044358253479, -0.5010558366775513, -0.730410099029541, -0.26492199301719666, -1.848785161972046, -0.4369930922985077, 1.1011390686035156, -0.25517815351486206, -1.7793980836868286, -0.24531038105487823, -0.5667499303817749, -0.7707213163375854, -0.9739289879798889, -0.669117271900177, 0.6509954929351807, -0.7677711248397827, 1.6378810405731201, 0.21803589165210724, -2.0148048400878906, -1.4725894927978516, 1.0573689937591553, -1.496757984161377, -1.5739511251449585, 2.6317005157470703, 1.751849889755249, -0.40057241916656494, 1.6504629850387573, -0.7013322710990906, 1.7457448244094849, -0.1854398548603058, 0.3553764224052429, 1.00471031665802, 0.6702052354812622, 0.03906994313001633, -0.40140190720558167, -1.2946792840957642, 0.9340654611587524, -1.3362141847610474, -1.3566436767578125, -0.07266481965780258, -0.9524196982383728, -0.026570262387394905, 1.5841199159622192, -2.2508342266082764, -0.5137391686439514, -0.23553819954395294, -0.8838974237442017, 0.25057080388069153, 1.3047959804534912, -1.2358850240707397, 0.5652929544448853, -0.6948463320732117, 0.7218424677848816, 0.3157176971435547, 1.161043643951416, 0.4041418731212616, 0.9808416962623596, -1.7628002166748047, 0.7528685331344604, 0.15244384109973907, 0.5327108502388, 1.415444016456604, 0.11309932172298431, -0.06344231218099594, 0.18435442447662354, 0.3316571116447449, -1.6153801679611206, -0.3256738781929016, -0.7338551878929138, 0.33461466431617737, 0.9850144982337952, -1.839236855506897, 0.2811789810657501, -0.3239779770374298, -0.2507355511188507, -0.6989415287971497, -1.0108087062835693, -0.448892205953598, -1.149340271949768, 0.3875041604042053, -1.1536226272583008, -0.00816455390304327, 1.3615881204605103, 0.062421008944511414, 1.7683161497116089, -1.0470978021621704, 0.7018139362335205, -0.4530915319919586, 1.6438896656036377, 0.11595416814088821, 0.453133225440979, 0.026291027665138245, -0.32862260937690735, -1.0614145994186401, -0.9033656716346741, -0.05271304026246071, 1.6556130647659302, -0.4935513734817505, 0.20147168636322021, 0.0008848276920616627, 0.28261592984199524, -0.2799413502216339, 0.3922511041164398, 0.946479320526123, -0.43948543071746826, -0.5997015237808228, -1.524779200553894, 0.2545756697654724, 1.100687026977539, -1.1243386268615723, 0.507604718208313, -1.015286922454834, 1.9302430152893066, 0.43394753336906433, -0.20418709516525269, 1.6218868494033813, 1.5849347114562988, 0.27583205699920654, -0.8148980140686035, -1.062360167503357, 1.1513715982437134, 1.39816153049469, -0.5046792030334473, 0.5734890103340149, 0.963456392288208, -0.25850191712379456, -0.05201397091150284, 1.7323216199874878, 0.5848047733306885, -1.1871943473815918, -0.04812905192375183, -1.57721745967865, 0.5504494905471802, -0.4706505239009857, -0.5635346174240112, -0.4245762526988983, 0.06091871112585068, -1.544708251953125, 0.13171043992042542, -0.8204823136329651, -1.9538886547088623, -0.007212894968688488, -0.9435106515884399, 0.2267240285873413, -1.4932984113693237, 1.6944406032562256, -1.0480166673660278, -0.2115667164325714, -0.45908546447753906, -1.3656178712844849, 0.18475976586341858, -0.273106187582016, -0.48150667548179626, 1.1963567733764648, -1.1654890775680542, -1.5813813209533691, -0.8157323002815247, 0.5132503509521484, 1.2800167798995972, 0.31548362970352173, 0.08374860882759094, -0.09128350764513016, 0.23265543580055237, 0.89250248670578, -2.8723645210266113, -0.8594179749488831, 0.05422728881239891, -0.30430305004119873, 2.1719443798065186, -1.0518323183059692, -0.7256535887718201, 1.0882279872894287, 0.49151745438575745, 0.8671954274177551, 1.536389946937561, 1.2836309671401978, -0.478039026260376, 1.4281320571899414, 2.0501155853271484, 0.037846874445676804, 0.5919142365455627, -1.1258724927902222, -1.1354610919952393, 0.6124895811080933, -0.6858066916465759, 0.39794954657554626, -0.14889150857925415, 0.6249262690544128, 0.7679793238639832, -1.2644999027252197, -1.4186303615570068, 0.27734601497650146, 0.3425191640853882, 0.020185744389891624, -0.14144080877304077, -0.6280799508094788, -0.08860397338867188, -0.6345564723014832, 0.5682106018066406, -0.05277588218450546, -0.018542105332016945, 2.2000582218170166, -0.4042501151561737, -1.5184451341629028, -0.04051589220762253, -0.538057804107666, -0.6199215650558472, -0.44607117772102356, -0.3611852526664734, -0.15702100098133087, -0.3616955280303955, -1.1742148399353027, -0.38358980417251587, 0.19682534039020538, 0.6305310726165771, 1.013053297996521, -0.6785938739776611, 0.3985733389854431, 0.5455343723297119, -0.5522795915603638, -0.7524938583374023, 1.1796317100524902, -0.7367955446243286, 1.0227504968643188, -0.1237124353647232, -0.5231980681419373, 0.8526924848556519, 1.3461169004440308, 0.2891434133052826, 0.20531779527664185, -1.995587944984436, -0.0563388466835022, 1.2065397500991821, -0.6961758136749268, 1.4617443084716797, 0.5925524234771729, -1.472674012184143, -0.3856077194213867, 0.04373819753527641, 0.2632509768009186, 0.3094489872455597, -0.1776198297739029, -0.7615205645561218, -0.08858296275138855, 0.3116486370563507, -0.12724314630031586, 1.2389031648635864, -0.27638015151023865, 0.29814913868904114, -1.298644781112671, -0.4040869474411011, -2.4128258228302, 2.9940028190612793, 1.1058475971221924, 0.7859205603599548, 0.4713252782821655, 0.41491758823394775, -0.029064973816275597, 0.7691761255264282, -0.11179250478744507, -0.2897246778011322, 0.4859531819820404, 0.5182814598083496, 0.648984432220459, -0.5574877858161926, 1.1740000247955322, -0.3224339783191681, -0.5111392736434937, 0.7427508234977722, -1.0429697036743164, 0.19321861863136292, -0.5279240012168884, -0.8483585119247437, 1.3440399169921875, 0.35698992013931274, 1.5544512271881104, -3.0492310523986816, -1.6650028228759766, 0.9213014841079712, 1.3203250169754028, 0.12474025040864944, -1.8574062585830688, 0.19299988448619843, 0.33256998658180237, 0.14420369267463684, -0.5240490436553955, -0.46186497807502747, 0.4055953323841095, 2.217477321624756, -0.8823539018630981, -0.3351759910583496, 0.6980709433555603, 0.7618764042854309, 1.0931488275527954, 0.9097055792808533, -0.05029686167836189, -0.6851878762245178, -0.4864301383495331, -0.3066488206386566, -1.4450445175170898, -0.30464547872543335, -0.5368209481239319, 0.19383804500102997, -0.21535615622997284, 1.0546659231185913, 1.9257558584213257, -0.6148634552955627, -1.6611626148223877, -0.1969919055700302, -1.0677014589309692, 0.11510463804006577, 1.718638300895691, -0.1478148251771927, 1.3667200803756714, -0.23022355139255524, 0.7322241067886353, -0.31656786799430847, 0.16171090304851532, -0.2658891975879669, -0.5833511352539062, -0.7402155995368958, 0.8034711480140686, -1.8815006017684937, 0.460496723651886, -0.6504440307617188, 1.422794222831726, 1.1457428932189941, 0.7145237922668457, -2.239293098449707, -0.9386631846427917, 1.5062587261199951, 0.3356887400150299, -1.5905468463897705, -1.2319607734680176, -0.5377016067504883, 0.8255385160446167, 0.8676725625991821, -1.5619608163833618, 0.3171754777431488, -1.7135401964187622, -0.7150855660438538, -0.9660124778747559, 0.4768761992454529, 1.1605395078659058, -0.08396096527576447, -0.18034207820892334, -0.5433163046836853, 0.7018530964851379, 1.0612258911132812, 1.0917788743972778, 1.4215857982635498, 1.2958080768585205, -0.45310458540916443, 1.3890453577041626, 0.8282637000083923, -1.0576248168945312, 1.1530603170394897, 2.5015625953674316, -1.437030553817749, -0.16616354882717133, -0.5282045602798462, 2.415419340133667, 0.28833404183387756, 1.4568558931350708, -1.4074132442474365, -0.10636068135499954, 0.37431401014328003, -2.3209176063537598, 0.8968703746795654, -0.9090180397033691, -0.6903402805328369, -1.1403083801269531, 2.5356690883636475, 1.9642596244812012, -0.5283979773521423, 1.5969122648239136, 1.8982843160629272, -0.20542079210281372, 0.16314290463924408, 0.10168953239917755, -1.5225605964660645, 0.8088787794113159, -1.8330706357955933, 1.7831321954727173, 0.5159493088722229, 0.11562082916498184, 0.03305093199014664, -0.7351472973823547, -0.34855103492736816, 1.624650001525879, 1.1300699710845947, 1.5387789011001587, -0.28776952624320984, 0.6545646786689758, 1.167357087135315, -0.5820440053939819, 0.24418018758296967, -0.6418851613998413, 1.1414258480072021, -1.4563586711883545, -0.468501478433609, 0.009009058587253094, -0.3499367833137512, -0.3707493245601654, 0.6361979842185974, 0.886692225933075, 0.33126917481422424, -0.7834882736206055, -0.5381639003753662, 0.8531529307365417, -0.37190383672714233, -0.49860522150993347, -0.454604834318161, 0.18329347670078278, 0.37704703211784363, 1.8082551956176758, -0.12115342915058136, 0.4806080758571625, 0.9881883859634399, -0.8837894797325134, 1.2782973051071167, -1.0767525434494019, 0.9887375235557556, 0.5709531903266907, 0.07195114344358444, 0.1695416420698166], [0.40097469091415405, -1.847435712814331, -1.3539198637008667, -0.1357363611459732, -0.8103162050247192, -0.5342683792114258, -1.1534643173217773, -1.3460338115692139, -0.08870598673820496, 0.0498206689953804, -0.6386463046073914, 0.6623319387435913, 0.6556603908538818, 0.7859649658203125, -0.2093518078327179, -0.5982308387756348, -0.9751598834991455, -0.5675485730171204, 0.3814844489097595, -0.04039205610752106, 1.960060477256775, -0.20323807001113892, -0.17433542013168335, -0.38637012243270874, -0.006088975351303816, -1.0792495012283325, -0.4383830726146698, -0.3423057198524475, 0.6779482960700989, 1.167412281036377, 0.18478479981422424, -1.8143408298492432, -0.2425355315208435, 1.3856513500213623, -0.10336332768201828, 0.03650211915373802, -1.9283134937286377, -1.146801471710205, -2.2871129512786865, 0.08661246299743652, 1.1137913465499878, 0.9322709441184998, -1.3764348030090332, -0.11416685581207275, 0.6951842904090881, 0.5628137588500977, 0.5774708986282349, 0.5900641679763794, 0.10364416241645813, -1.3811460733413696, 0.008377375081181526, -0.13140766322612762, 1.038960576057434, 1.9032065868377686, 0.7537388801574707, -0.23822109401226044, -0.1310311108827591, -0.4516051709651947, -1.5645434856414795, -0.7174042463302612, -1.658233880996704, -0.1952817589044571, -1.8759757280349731, 0.425491601228714, 0.2936440110206604, -0.9835094213485718, -2.1815922260284424, 1.1709275245666504, -3.2744901180267334, 0.8871188759803772, 1.8130801916122437, -1.0146641731262207, -0.047498710453510284, -0.18388721346855164, -0.7743898034095764, 0.2573585510253906, -2.2628026008605957, -0.6825293302536011, 1.5557061433792114, 0.4231564402580261, 2.569385528564453, 0.31920287013053894, 1.3115140199661255, 2.4504504203796387, 0.7226472496986389, 0.24139641225337982, 0.5499482750892639, 0.10208582878112793, 0.0024008448235690594, -0.9552030563354492, 1.9565874338150024, -0.928320050239563, 0.14327402412891388, -0.2308894693851471, -1.782402515411377, 0.45689520239830017, 0.8210524916648865, -0.47212278842926025, 0.8025510907173157, 0.9159857034683228, 1.0566034317016602, 0.1701750010251999, 0.44698822498321533, 0.36071693897247314, -0.8522779941558838, -0.888911247253418, -0.9937688112258911, 0.5900774002075195, 0.7385435700416565, 1.2984585762023926, 0.7790795564651489, -0.3461354374885559, 1.8146183490753174, -0.7056974172592163, 2.3213326930999756, -0.35754600167274475, -0.20378543436527252, -0.2553566098213196, -0.4717473089694977, -0.06928816437721252, 0.14673367142677307, -1.395706057548523, -0.08042984455823898, -0.4650433659553528, -0.26246362924575806, -0.49730587005615234, -0.014153599739074707, 0.41796639561653137, -0.21769045293331146, -0.6328072547912598, 1.968103289604187, 1.3253215551376343, -1.009203314781189, 0.5730755925178528, -0.42096081376075745, -1.2624765634536743, 0.1671493947505951, 0.6561166048049927, -0.7028771638870239, 0.3693476915359497, 0.763359010219574, 0.04569574445486069, 0.7228307723999023, 0.6818366646766663, -0.4191884696483612, 0.8625044822692871, 1.5672436952590942, -0.10620435327291489, -0.3578444719314575, -0.16089782118797302, -1.9883325099945068, 1.142279863357544, 0.8919044733047485, 0.3751540780067444, -2.2770726680755615, -0.6204289793968201, -0.6448021531105042, 0.8775901198387146, 1.484671711921692, -1.4587393999099731, -0.14456702768802643, 0.5081683397293091, 1.6599953174591064, 0.5587791204452515, -0.0718345195055008, 0.5257080793380737, -1.0886191129684448, -0.2865389287471771, -0.6696028113365173, 0.21244291961193085, -0.47365039587020874, 0.04946425184607506, -1.0695358514785767, 0.03472991660237312, 1.3061721324920654, -1.4679622650146484, 0.482686847448349, 0.49459704756736755, 0.6858492493629456, 1.0202778577804565, -0.6390907764434814, 1.254793643951416, -0.3387907147407532, 1.2007012367248535, -0.47894394397735596, -1.7259423732757568, -0.13836076855659485, 0.02320178784430027, 0.1844109296798706, 1.8283330202102661, 2.374171733856201, 1.269105076789856, -0.8498430252075195, 1.8819301128387451, 1.2404797077178955, 0.993632972240448, 1.5485867261886597, -0.33335641026496887, -0.1853480339050293, 0.5693660378456116, -0.9024481177330017, 0.34941115975379944, 0.9463838338851929, 0.39440301060676575, -0.9143736362457275, -0.1963748335838318, -0.1464744210243225, -1.114047646522522, -1.5120623111724854, 0.2257828414440155, 0.10310575366020203, 1.055733323097229, 0.26936742663383484, 1.3226221799850464, 0.8577879667282104, -1.226146936416626, -0.550774335861206, -0.15415175259113312, 0.46982601284980774, 0.15661780536174774, 0.5784059166908264, -1.404506802558899, 0.9541361927986145, 0.3223188817501068, 0.5717951059341431, 0.2612464129924774, 1.567609429359436, -1.2164231538772583, 0.9461120963096619, 1.117011547088623, -0.706851601600647, 0.9505046010017395, 1.4137952327728271, 0.18586918711662292, -0.9430649280548096, -0.17724238336086273, 0.07906132191419601, 0.2539098858833313, -0.5720933675765991, 0.967413067817688, -1.1440672874450684, -0.531126856803894, 0.9045716524124146, -0.14972171187400818, -1.3955299854278564, -0.2716272473335266, -0.6867768168449402, 0.5843239426612854, 1.1766127347946167, 0.33783286809921265, -0.39502477645874023, -0.8807882070541382, 0.11579450964927673, -0.7814978361129761, 0.8871289491653442, -0.11960747092962265, 1.268835186958313, -1.0331093072891235, 0.630230724811554, 0.7249488234519958, 1.8438050746917725, 0.2748256325721741, -0.5072131156921387, 0.2128370851278305, 0.08700411021709442, 0.7304851412773132, -1.320923089981079, 0.7721738219261169, 0.35745757818222046, 1.1238406896591187, -0.09161899238824844, 0.16591013967990875, -1.2021650075912476, -1.5463001728057861, 0.3682023882865906, -0.17876850068569183, -0.6488157510757446, -1.930001139640808, -0.3327190577983856, -0.6480181813240051, 0.026000220328569412, 0.7712462544441223, 1.7408109903335571, 0.9280327558517456, -1.1696548461914062, -2.002319574356079, 1.9733268022537231, 1.3691978454589844, -0.5230116844177246, 1.707319736480713, -1.1433802843093872, -0.217844158411026, -1.9478328227996826, -0.17699690163135529, -0.9985218644142151, 0.6349549293518066, 0.7205275297164917, -0.5870364904403687, 0.4410703182220459, 0.3121061325073242, -0.0194545891135931, 0.5340055227279663, -0.9314376711845398, -0.5095782279968262, 0.26830631494522095, 1.385500192642212, -1.1150026321411133, 0.3847883343696594, 1.412628412246704, 0.13347211480140686, -2.012850046157837, -0.8445324897766113, -0.48673033714294434, -0.6505051255226135, -0.5923654437065125, -0.11362773180007935, -0.879462480545044, 1.0842868089675903, -0.6042198538780212, 0.8120744228363037, -0.3376082479953766, 1.217504620552063, 0.05363258346915245, -1.6836390495300293, 2.03800892829895, -0.6061850190162659, 0.4635803997516632, 1.4204097986221313, 0.16939249634742737, -0.8881286978721619, -2.27596116065979, -1.2604596614837646, 1.0816200971603394, 0.5460999011993408, 1.4426847696304321, 0.7883576154708862, -1.4609297513961792, -0.7029662132263184, -0.2682291269302368, 0.22999627888202667, 0.005383598152548075, 0.05438517779111862, 2.1354312896728516, -1.813987374305725, 1.2436463832855225, 1.7504198551177979, 0.7249513864517212, 0.019786907359957695, -0.2157789021730423, 0.15400545299053192, 0.4060143232345581, 0.30977797508239746, -1.363065481185913, 0.8140509724617004, 0.17753884196281433, -1.5299268960952759, -0.2401251196861267, -0.3137999176979065, -0.9372947216033936, -0.17545217275619507, 1.8397958278656006, 1.7164140939712524, 0.8504317998886108, -1.1502727270126343, -0.2725651264190674, 0.17966049909591675, 0.14129838347434998, -0.6524964570999146, -1.2929248809814453, 0.3998486399650574, 0.13127431273460388, 0.33548104763031006, 0.12715086340904236, 0.4658159911632538, -0.03695795312523842, 0.4658903181552887, -1.1782422065734863, -1.469955563545227, -0.3705461323261261, -1.0484764575958252, -1.2785117626190186, 1.1922500133514404, -0.4658333659172058, 0.4848025143146515, 0.7078414559364319, -0.3357287049293518, -0.01116577722132206, -2.3407657146453857, 1.3627015352249146, 0.3190830647945404, -1.221523642539978, -0.6994436979293823, 0.8633442521095276, -0.7005369663238525, 0.7865275144577026, -1.9784398078918457, 0.9659916162490845, 0.06151445582509041, -0.3915504813194275, 0.392553448677063, 1.3144807815551758, -0.7869659662246704, 0.3582571744918823, 0.442516952753067, 0.8567978143692017, 0.7789140939712524, -0.3766767680644989, -0.10436704754829407, 0.16682715713977814, 1.2304271459579468, -1.1581801176071167, -1.0183318853378296, -0.8873576521873474, -0.6222658157348633, 0.5881041884422302, 1.12208092212677, -0.9636226892471313, 1.2022452354431152, 1.0029176473617554, -0.2674325704574585, 1.4269770383834839, -1.0187263488769531, -0.23202288150787354, -0.012896898202598095, 1.4230785369873047, -1.8659414052963257, -0.3576095998287201, 1.7032396793365479, -0.08863727003335953, 0.00950927659869194, -0.11792276054620743, -1.7864400148391724, 0.3658001124858856, 0.989112377166748, -0.8893575668334961, -0.34246188402175903, -0.4087989330291748, -0.472871333360672, 2.7680108547210693, -0.5911054015159607, -0.9856207370758057, 0.5517484545707703, -0.309175580739975, -1.178154706954956, 0.062332846224308014, 0.5334135890007019, 0.6512893438339233, -0.19364158809185028, 0.38219332695007324, -1.2274761199951172, -0.5714280605316162, -0.3046581447124481, 1.692279577255249, -0.07929550856351852, 0.23577430844306946, 0.3008292019367218, 1.3262052536010742, -1.136543869972229, 0.010853073559701443, 0.005282540339976549, 0.08447014540433884, 1.0667901039123535, -1.8858011960983276, 0.9745416641235352, 1.553499460220337, -0.2709548771381378, -0.31071123480796814, 0.031125493347644806, 0.20718643069267273, 1.7742280960083008, -1.8216105699539185, 1.5566973686218262, -0.02394828200340271, -0.5624169111251831, -0.4324344992637634, -1.3310043811798096, -0.46129274368286133, 1.05144202709198, -1.0312602519989014, -2.3483479022979736, 0.5477317571640015, -0.47898513078689575, 0.5119223594665527, -1.8868248462677002, -0.18516185879707336, -1.7718079090118408, -1.1326321363449097, -0.11131584644317627, 0.9252709746360779, -0.17254869639873505, 0.2808559834957123, 1.0780644416809082, -1.7410238981246948, 0.07736837863922119, -0.6990236639976501, 0.34232231974601746, 0.4639510214328766, -1.3949079513549805, 0.4409324526786804, 0.30074542760849, 1.126137614250183, 0.5714470148086548, 0.8719927668571472, 0.06386516988277435, -1.0871297121047974, 0.36844784021377563, -0.22226545214653015, 0.24848638474941254, 0.8698693513870239, 1.6052470207214355, 0.571180522441864, 1.111830711364746, 1.4039288759231567, 0.4098559617996216, -0.48380592465400696, -0.011509530246257782, -1.1047922372817993, 0.241936594247818, 0.7278138399124146, -0.6657859683036804, 0.17075563967227936, -0.532080888748169, 1.1754686832427979, -0.734165370464325, -1.5260692834854126, -1.3056433200836182, -0.07695195823907852, -1.2070847749710083, -0.10227043926715851, -1.0122077465057373, -0.27332085371017456, -0.8294552564620972, 0.822939395904541, -1.209756851196289, 2.1764917373657227, 0.02125278115272522, 1.7444952726364136, -0.20800548791885376, 2.5551788806915283, -0.7579319477081299, 0.8330135941505432, -0.06374562531709671, 1.0625102519989014, -0.8417258262634277, 0.6903514862060547, -0.013940354809165001, 0.11543109267950058, 1.8806966543197632, -0.10520142316818237, 0.7046653628349304, 0.09692699462175369, 0.8237718343734741, 0.8678525686264038, 0.3070613741874695, 1.1960629224777222, 0.9770752191543579, 0.07614892721176147, -2.5592544078826904, 1.0315734148025513, -0.001204191124998033, 0.8687754273414612, 0.6778147220611572, 0.8069542050361633, 0.7838097810745239, -0.8961957097053528, 0.6597112417221069, 0.44283929467201233, -0.31651031970977783, 0.15122036635875702, 0.39787793159484863, -0.7300472855567932, -0.36488285660743713, -1.0983080863952637, -0.8775525093078613, 1.04971182346344, -0.27499955892562866, 0.05027264356613159, 1.7777389287948608, 2.473214864730835, -1.4476943016052246, -0.7255716919898987, 0.22030574083328247, 2.7667548656463623, -1.1544501781463623, -1.6813596487045288, 0.11156265437602997, -0.05281227082014084, -0.7831876873970032, 0.053462572395801544, -0.2965802252292633, -1.3647292852401733, 0.4726603329181671, -0.4030752182006836, 1.4507756233215332, -0.053744684904813766, -1.1495780944824219, -0.5032036900520325, 1.2344413995742798, -0.17470794916152954, -0.2936396300792694, 0.9171932935714722, 1.1889772415161133, 1.4866907596588135, 1.8868553638458252, -1.5385537147521973, 1.5003366470336914, 0.14738331735134125, 1.9095019102096558, -0.5858043432235718, 0.8633201122283936, -0.19928500056266785, 0.4621325135231018, -0.5887993574142456, -0.6921005249023438, 0.25514286756515503, -0.4735962450504303, 0.6616275310516357, -1.0178128480911255, 0.8530194163322449, 2.1367549896240234, 0.6303310990333557, -0.7707723379135132, 0.31540897488594055, -0.013187546283006668, 0.8527811765670776, 0.25693297386169434, -0.24702519178390503, 1.1562806367874146, -0.5844938158988953, -0.12647174298763275, 0.7353927493095398, -0.7005088925361633, -0.4683285355567932, -0.23412708938121796, -0.8892836570739746, -0.5357480645179749, 1.6993286609649658, 0.12404301017522812, -0.9372504353523254, -0.961843729019165, 1.1851532459259033, 1.1975148916244507, 2.090435266494751, 0.34772175550460815, -1.190915584564209, -0.21907760202884674, 0.354878693819046, 0.985205888748169, 0.1971563994884491, 0.31568121910095215, 1.1821056604385376, 0.25645166635513306, 1.2423359155654907, 0.5191553831100464, -0.39600062370300293, -1.021515965461731, 1.910326361656189, -2.3566136360168457, -0.2149811089038849, 2.570502996444702, 1.4285014867782593, 0.8702058792114258, -1.4680501222610474, -0.05822309851646423, 0.33251485228538513, 0.7943596839904785, -2.361778974533081, -0.650307834148407, -0.1995425820350647, 0.4013501703739166, 0.31086716055870056, -1.3800487518310547, -0.44857102632522583, -1.7717077732086182, 1.7659671306610107, 0.27240243554115295, 0.24125316739082336, -0.9925662875175476, 0.3386250436306, 1.6193135976791382, -1.0491912364959717, -1.354414463043213, -0.016411608085036278, -1.5909980535507202, -1.2669527530670166, -1.6404350996017456, 0.9511212706565857, -1.1375192403793335, -0.39257973432540894, -0.6451063752174377, 1.3251149654388428, 0.6053317189216614, -1.207554817199707, -0.31836920976638794, -0.7203599810600281, -0.1008673831820488, 1.3106297254562378, 0.8055686354637146, -0.14285635948181152, 1.7288278341293335, 1.772202491760254, -2.1090712547302246, 0.31893789768218994, 0.7473299503326416, 1.139237880706787, -0.14836052060127258, -0.7366706728935242, 1.5227922201156616, -0.10468937456607819, 0.1838354617357254, -1.2426022291183472, 1.496427297592163, -1.1695672273635864, -0.2013407200574875, 1.325911283493042, 0.6590368151664734, -2.3183391094207764, 0.8259611129760742, -1.174544334411621, 1.999474287033081, -0.6787278056144714, 1.010398507118225, -0.831548273563385, 3.2681686878204346, 1.0496551990509033, 0.0524434968829155, -1.4911096096038818, 1.8639121055603027, 1.1953206062316895, -0.6466176509857178, -0.7997339367866516, 0.9635571837425232, 2.5469064712524414, 0.1630072146654129, 2.613921880722046, -0.25027576088905334, -0.08792667090892792, -0.10625562071800232, 0.8345057368278503, -0.4642738997936249, -1.1700615882873535, -0.4597894549369812, 0.26257747411727905, -0.24041961133480072, 1.4413036108016968, 0.19786964356899261, 0.8781869411468506, -1.2455140352249146, -0.8625131249427795, 0.7346513271331787, -0.7164794206619263, -1.2513865232467651, -0.07392877340316772, 0.7409714460372925, 0.8465184569358826, 0.015809573233127594, -0.034431323409080505, -0.693299412727356, 0.027323488146066666, -0.45501288771629333, -0.56785649061203, 0.48792174458503723, 0.9981936812400818, 0.4809179902076721, -2.1823997497558594, -0.902606189250946, -0.747947633266449, -0.7225383520126343, 1.1554152965545654, 0.5582703948020935, 0.2897019684314728, -0.20577354729175568, 0.7542818784713745, -0.6785310506820679, 0.39368826150894165, 0.4067305624485016, 0.7055584192276001, -1.224312424659729, 0.8046392202377319, 0.2697583734989166, -0.630192756652832, 0.42772120237350464, 0.06629599630832672, 1.097737431526184, -1.2203179597854614, -1.7829257249832153, 1.6410670280456543, -1.042129635810852, 0.3238098621368408, 0.15180917084217072, -0.8799448013305664, -1.7956396341323853, 1.5300507545471191, -2.8842668533325195, -1.2887009382247925, -1.5243228673934937, 0.7538391351699829, 0.36510157585144043, -0.8272484540939331, 1.4306349754333496, -0.7941817045211792, 1.5045186281204224, -1.201220154762268, -0.8102702498435974, 0.39774391055107117, 1.600312352180481, -0.1905660480260849, 0.002959097735583782, 0.9524161219596863, -0.5193025469779968, -0.7661958932876587, -0.4403037428855896, -0.16337688267230988, 0.7255859375, 1.0126168727874756, -1.455654263496399, 0.24492931365966797, -0.4177689850330353, -0.028118517249822617, -0.6420842409133911, 1.534143328666687, 0.1412643939256668, 0.08933531492948532, -1.212306261062622, 0.6197423338890076, 1.1615681648254395, -0.24263238906860352, 0.0651162713766098, 0.8655267357826233, -1.2479830980300903, -0.2072889506816864, 0.4835937023162842, 1.1075464487075806, -0.9272027611732483, -0.047239795327186584, 0.8759022951126099, 2.2229185104370117, -2.3306021690368652, -0.22666415572166443, -0.7824850082397461, -1.426595687866211, -0.45379331707954407, 2.342115640640259, 0.5787918567657471, -1.6044731140136719, 0.9031794667243958, 0.7803446650505066, -0.8521198630332947, 2.438047170639038, -0.3291901648044586, -0.14095649123191833, 0.09459756314754486, -0.0807240828871727, -0.23307786881923676, 1.0857117176055908, -1.2259535789489746, -0.7470132112503052, -0.6149498224258423, -0.2674184739589691, -0.6845077276229858, -1.034843921661377, 0.6351592540740967, -0.4387744665145874, 1.5040212869644165, 0.47117501497268677, 1.5237468481063843, 0.2676217257976532, -0.1020439937710762, -1.052130103111267, -0.04899299144744873, 1.2543439865112305, -2.2481086254119873, -0.09245967119932175, -0.2630916237831116, 0.8201330900192261, -0.48118793964385986, -0.5395132899284363, 0.2590722441673279, 1.7696142196655273, 1.0626636743545532, 0.029143720865249634, 0.08302000164985657, 0.10780200362205505, -1.2328886985778809, -0.6498956084251404, -2.2905337810516357, -1.6580296754837036, 1.596724271774292, -0.26588407158851624, 0.13252320885658264, 0.34128043055534363, 0.6577730774879456, -0.3257893919944763, 1.526710033416748, -1.410609245300293, -0.25719088315963745, -0.31829506158828735, -1.2236804962158203, -2.214632749557495, 0.4185909926891327, -0.05377182364463806, -0.9694523215293884, 0.026554957032203674, -0.016330359503626823, -0.1243242546916008, -0.16541346907615662, -0.41781753301620483, -1.0415666103363037, -0.1693073809146881, -0.2950073778629303, -0.19672371447086334, 1.4888596534729004, 0.8546295166015625, -0.2597123384475708, -0.25202012062072754, 1.1485642194747925, 0.9552196264266968, -0.35858097672462463, 0.6725996732711792, -0.34376680850982666, -1.7367252111434937, -1.211631178855896, -0.8409102559089661, 0.8251405954360962, 0.8811870217323303, -0.8953311443328857, -0.1642448753118515, -0.8155817985534668, -0.28214335441589355, 1.1314531564712524, -0.4359460473060608, -2.17513370513916, -0.9663355946540833, -1.5126532316207886, -0.5713685154914856, -0.05813547223806381, 1.9489893913269043, 0.10729558765888214, 1.7680506706237793, 0.13569006323814392, 0.10966996848583221, -0.32964691519737244, -1.4638628959655762, 1.3047764301300049, -0.41182419657707214, 0.41151702404022217, -0.11369743198156357, -0.06555869430303574, 0.7833410501480103, -1.125617265701294, -1.840407371520996, -0.5086259841918945, 0.6367419362068176, -0.1448633372783661, -1.055152177810669, 0.40754544734954834, 0.41561245918273926, -0.34237903356552124, -0.6097890138626099, -0.721576452255249, -0.05201129615306854, 0.6287444233894348, -1.6118993759155273, -0.3192315101623535, -1.000478744506836, 1.6636900901794434, 0.06871362030506134, -1.2327309846878052, -0.9067776203155518, 0.4872322976589203, -0.8581846952438354, -0.4975232481956482, 0.7500845193862915, 1.3000091314315796, -0.17914564907550812, 2.1671581268310547, -0.2414468675851822, -0.36586618423461914, -0.5448182225227356, -0.39549028873443604, 0.7602397799491882, -0.6850574016571045, -1.052096962928772, 1.3689830303192139, 0.6548721194267273, 0.06112939119338989, 1.2257276773452759, -0.3748530149459839, 0.2159690260887146, 2.459528923034668, -0.07571205496788025, 0.4412076771259308, 0.6755626201629639, -0.43201813101768494, -0.49413561820983887, 0.40743309259414673, -2.5233991146087646, -1.0887234210968018, 0.3145248591899872, -1.5679032802581787], [1.0784510374069214, 0.7991921305656433, 0.6978957653045654, -1.335747241973877, -0.287269651889801, -2.818596601486206, 1.8868434429168701, 1.1452912092208862, 0.3319898247718811, 0.25505927205085754, -0.1090254932641983, -1.2375551462173462, -0.27252325415611267, 0.1649588793516159, -0.5819490551948547, -0.1530124545097351, 0.8338139653205872, 1.415043830871582, 0.15431305766105652, -0.3310454785823822, 1.8777226209640503, -1.9487619400024414, -1.3751686811447144, 0.5983136892318726, -0.5252707004547119, 0.6512371897697449, -0.10739373415708542, -0.4475949704647064, 1.0869884490966797, 0.032306794077157974, 0.6635265350341797, 0.5989348292350769, 1.6756229400634766, -0.11590651422739029, -0.12097030878067017, 0.22951640188694, 1.3076668977737427, -0.8572896718978882, 0.11756826192140579, -0.9059399366378784, 0.941121518611908, 0.4446295201778412, 0.4085625112056732, -1.105562448501587, 0.09022674709558487, -2.622140884399414, -0.3499405086040497, 0.16874302923679352, -0.7887124419212341, -0.4809935688972473, -0.5731713771820068, 1.6726891994476318, 0.33120059967041016, 0.4750478267669678, 1.0143793821334839, -0.008105170913040638, -0.4805814325809479, -0.1680455356836319, 0.9002379179000854, -0.5530370473861694, 1.2557209730148315, 0.982088565826416, -1.3377209901809692, -0.2431478053331375, 0.07848411798477173, 0.3893650770187378, -0.5288052558898926, 1.5176697969436646, -0.902926504611969, 0.4667855203151703, -1.298743486404419, -0.46899664402008057, 1.0632712841033936, -0.35316044092178345, 0.1444004774093628, -0.3941689431667328, -1.2749409675598145, 0.4616248607635498, -0.4227634370326996, -0.7176706790924072, 1.0334895849227905, -2.0801992416381836, -0.9899066090583801, -0.09188058227300644, -0.3270873725414276, 1.0498905181884766, -1.513893485069275, 0.11698516458272934, -0.05321485549211502, 1.0616718530654907, 0.3300793170928955, -1.4724388122558594, -0.592505693435669, -1.2502026557922363, -0.9449618458747864, -0.2663489282131195, 0.4820016622543335, 0.37914493680000305, -1.1959691047668457, -1.724187970161438, 0.2311234176158905, -0.10515276342630386, -0.36018484830856323, -0.598162055015564, 2.130762815475464, 1.305930733680725, -0.2633976340293884, -0.2704252600669861, 0.8644351959228516, 0.5620707869529724, 0.5258893966674805, -0.764086902141571, -0.44293951988220215, -0.841782808303833, -0.24425898492336273, 0.24746105074882507, 1.0975476503372192, 1.6813362836837769, -0.5624770522117615, -0.6416207551956177, 1.0903756618499756, -0.3960750699043274, -0.34257012605667114, 0.7386184930801392, -0.7125055193901062, -1.810420036315918, 0.6163699626922607, -0.4332277774810791, 0.18760353326797485, -0.26136186718940735, -0.26256367564201355, 0.5545255541801453, 0.36100029945373535, 0.013600938953459263, -0.06844398379325867, 0.5177666544914246, -1.4735058546066284, -0.6300014853477478, -2.1716952323913574, 0.5855305194854736, -0.12568287551403046, -1.4431158304214478, -0.2205512523651123, 1.3649203777313232, -0.26791852712631226, 0.9161984920501709, -1.2099055051803589, 0.5918372869491577, 0.03202449157834053, -0.09105376899242401, -0.45192596316337585, 1.6604554653167725, 1.3483846187591553, -0.14680640399456024, 0.31709232926368713, 0.2518453896045685, -0.07356753945350647, 1.1241079568862915, 1.7500020265579224, 0.5362411141395569, 0.7285184860229492, -0.10088586062192917, 0.026012476533651352, -0.6613684296607971, 0.5998511910438538, -0.38078323006629944, 0.38226401805877686, 0.9976019263267517, -1.2675336599349976, -1.2877784967422485, -0.4816204309463501, -1.3588041067123413, -0.3899654746055603, -0.28298404812812805, -0.9567863941192627, -1.2431186437606812, 2.6135659217834473, 0.8003730773925781, 0.7759250402450562, -0.3146354556083679, -0.5192477703094482, 1.5408062934875488, 1.8579171895980835, -0.5159883499145508, -1.5539696216583252, -2.6741273403167725, -2.0221927165985107, -0.0233903918415308, -0.5381492376327515, -1.021759033203125, -1.2631868124008179, 1.768524169921875, 1.1367082595825195, -1.2501864433288574, -0.8490656018257141, 1.2855448722839355, 2.3559374809265137, -1.307024598121643, -0.6203823685646057, 1.6904311180114746, 0.6976683735847473, 0.3246552646160126, -0.7926210761070251, -3.8580145835876465, 0.8377923965454102, 0.45144766569137573, -0.32732826471328735, -1.8497565984725952, -1.3411593437194824, 0.9479410648345947, 1.3218071460723877, -0.7363241910934448, -0.4782921075820923, 1.0322540998458862, -0.5565536618232727, 0.6714609861373901, -0.8727023005485535, -0.2542192339897156, -0.8580556511878967, -0.9730688333511353, -2.1743404865264893, -1.214034080505371, -0.2540653944015503, -0.11555667221546173, -0.33568325638771057, -1.560246229171753, -1.3322099447250366, -1.4662253856658936, -0.3870331943035126, 1.7302190065383911, -1.1668323278427124, 0.6786954402923584, -0.6371580362319946, -0.5600716471672058, 0.8359249234199524, -1.576356053352356, -1.7267574071884155, 1.3333889245986938, 0.672618567943573, -0.0990133136510849, 0.941815197467804, -0.9370039701461792, -0.6153120994567871, 0.23997808992862701, 1.5267350673675537, -0.4886857271194458, -0.4009375274181366, 0.33074069023132324, 1.5980031490325928, 0.638735294342041, 0.6305623650550842, -0.1630246937274933, -0.18161681294441223, 0.05504021793603897, 0.4480273723602295, 0.5560941696166992, 0.14199475944042206, 0.7975977063179016, -0.4624764025211334, -0.1765851080417633, 0.514901340007782, 0.7862944602966309, -1.2536412477493286, -0.7949573397636414, -0.44088879227638245, 1.211979866027832, -0.3284832835197449, -1.3388715982437134, 1.2492448091506958, -0.9273935556411743, -1.423996090888977, 2.620872974395752, 0.6196911334991455, -0.34418970346450806, -0.03477639704942703, 1.0453011989593506, -0.3224446177482605, -0.33569252490997314, -0.33236485719680786, -1.2604947090148926, 2.5120418071746826, -1.8373552560806274, -0.9545185565948486, 0.15559396147727966, -0.5570327639579773, 0.865862250328064, -1.132741928100586, -0.2098686695098877, 0.22019940614700317, -0.011274571530520916, 0.4165700674057007, -1.3415720462799072, -1.8129726648330688, 0.39415886998176575, -1.3748304843902588, -2.219076633453369, 0.1185072660446167, 0.7244240045547485, 0.1534215360879898, -0.290100634098053, -0.48049408197402954, -0.1715264916419983, -0.40563899278640747, -0.3690424859523773, 0.2248954027891159, 0.27614161372184753, 1.1476045846939087, -1.4326974153518677, 0.06841832399368286, 0.2586337924003601, 1.24750816822052, 0.2527741491794586, -0.8686320781707764, 1.3709954023361206, -0.6885780096054077, -0.11583936959505081, -1.4839390516281128, 0.8467212915420532, 0.31731003522872925, 1.097078561782837, 1.4400027990341187, -0.15349456667900085, 0.40677979588508606, -0.9167298674583435, 0.5526453256607056, -0.4370354115962982, -0.28495529294013977, 0.36100971698760986, 0.06732603907585144, 0.5548655390739441, -0.2639508545398712, -1.556047797203064, -0.45199546217918396, 0.5719805955886841, 1.4228452444076538, -0.20979562401771545, 0.36938947439193726, -1.6387532949447632, 0.7407510876655579, -1.0255990028381348, -0.22113612294197083, 0.2791156470775604, 0.9699791669845581, 1.6056904792785645, -0.23695547878742218, 1.309086561203003, -0.2982412576675415, -0.7100424766540527, 0.21873052418231964, 0.4942021071910858, -1.0258574485778809, -0.5915423631668091, 0.3731672763824463, -0.17020320892333984, 1.5671212673187256, 1.0094096660614014, -0.41793134808540344, 0.4723691940307617, -0.864765465259552, -1.474584937095642, -1.013981580734253, -1.772340178489685, -0.12903402745723724, 1.0547014474868774, 0.6751242280006409, 1.606561303138733, 0.0644102469086647, -0.3758912682533264, -0.499186247587204, 0.4356420636177063, -0.9385013580322266, 0.6251951456069946, 0.7467008829116821, 0.9142126441001892, 0.19347554445266724, -0.11936673521995544, -0.9992434978485107, 2.4639408588409424, 0.0777682512998581, -0.4535636901855469, 2.875425100326538, -0.5681242346763611, 1.2204153537750244, 1.247269868850708, 1.0109210014343262, 0.8463128209114075, -0.5247541069984436, -0.1745733767747879, -0.2825031578540802, -0.5629675984382629, 0.3987809717655182, 0.6823137402534485, -0.40287551283836365, 0.6782243251800537, -1.275475263595581, 0.725010871887207, -1.616767406463623, 1.1629105806350708, -0.04997559264302254, -0.6872217655181885, 0.3725147843360901, 0.24586480855941772, 0.8525930047035217, 0.3298909366130829, 0.6556268930435181, -1.2234138250350952, 0.4230719804763794, 1.7146596908569336, -0.8073396682739258, 0.6234700679779053, 1.3077588081359863, -2.157215118408203, 1.9954969882965088, -1.206986665725708, 1.4930284023284912, 0.16879230737686157, 0.26520851254463196, -0.5556225776672363, -0.4851975440979004, 0.11237434297800064, 0.6499543190002441, 0.4817877411842346, 0.2715439200401306, 0.9476683735847473, 0.15409919619560242, -0.2816934585571289, 1.048236608505249, 0.5124945640563965, -1.729303002357483, -1.4279228448867798, 1.4109045267105103, -1.398507833480835, -0.48454663157463074, 1.2457654476165771, 0.18577665090560913, -0.1805577129125595, -0.9542533159255981, -1.2882269620895386, -0.9683015942573547, 0.5273758769035339, 1.1019691228866577, -0.13409030437469482, 0.8933281898498535, 0.3717651069164276, -0.25996026396751404, 0.28648582100868225, 0.033320095390081406, -1.262998342514038, 0.32470864057540894, 1.313332438468933, -0.11023566126823425, 0.4175333082675934, -0.0809088870882988, -0.657758891582489, -2.0822386741638184, -0.34440886974334717, 0.648453414440155, -2.8098766803741455, -1.9596513509750366, 0.8076561093330383, -1.1458473205566406, 0.31517693400382996, 0.37864717841148376, -0.5424290895462036, 0.502762496471405, 1.2659825086593628, 1.6847091913223267, -0.2232547402381897, 0.004045980051159859, 1.5531084537506104, -2.0993261337280273, 0.003561720484867692, -1.2816333770751953, 1.4390262365341187, -0.03257149085402489, 0.6851922273635864, 0.27649158239364624, -1.425117015838623, -0.7331514954566956, 1.6778000593185425, -0.13188059628009796, 0.6767988801002502, 0.21127989888191223, 0.031322184950113297, -1.494545578956604, -0.4331617057323456, 0.3254479467868805, 0.003693448379635811, -0.47624674439430237, -0.4902972877025604, 0.23444268107414246, 0.6679388284683228, 0.23855268955230713, -0.7918826937675476, 0.08552423864603043, 1.2424101829528809, 0.7909477353096008, 0.8186451196670532, 0.8834939002990723, 0.5234646201133728, 1.5988579988479614, -1.1510310173034668, -0.8852809071540833, -0.41303426027297974, 0.39029252529144287, -0.5636803507804871, -1.3035657405853271, 0.2004171907901764, 0.7666659951210022, 0.2278061956167221, -0.49125105142593384, -1.8662368059158325, 0.2808057963848114, -0.6052228212356567, 0.11592863500118256, 0.7033218741416931, 1.1384015083312988, 0.2338939756155014, -1.111465573310852, -1.4354597330093384, 0.22502532601356506, 0.052919067442417145, 0.23005586862564087, 0.6261300444602966, 1.3610974550247192, -0.32971519231796265, -2.7584381103515625, 0.15625318884849548, 0.7104386687278748, 0.45760220289230347, 0.8784875869750977, -1.6745615005493164, -0.70449298620224, -1.930677056312561, -1.2914918661117554, 1.1750541925430298, -0.6417241096496582, -1.8271788358688354, -0.3088666796684265, -0.22609210014343262, 1.564987301826477, -0.9942742586135864, 1.7127236127853394, -0.812287449836731, -0.4296739399433136, -0.3704548180103302, 0.15656955540180206, -0.8184484243392944, 0.3907223641872406, -0.6627305746078491, 0.9282946586608887, 0.6834993958473206, -0.3075072765350342, -1.6214245557785034, 0.3233512341976166, -0.6888303160667419, -0.6276222467422485, 0.5824449062347412, 0.27757009863853455, 0.5363104939460754, 1.132789969444275, -0.21668046712875366, 0.2762782871723175, -1.5468670129776, -0.9552311301231384, 1.0538434982299805, 1.0353527069091797, -0.9028458595275879, 0.8121612668037415, -1.2358918190002441, -1.7332127094268799, -1.9686321020126343, -0.24476072192192078, 2.759706497192383, 0.8877031207084656, 0.006349568720906973, 0.4903826415538788, 0.0007312198868021369, 0.3642219603061676, 0.182756245136261, 0.07098716497421265, 0.9091426730155945, 0.6214998960494995, -1.9449045658111572, -2.1391851902008057, 1.3981913328170776, -1.0342518091201782, -1.2096625566482544, -0.28387320041656494, -0.48468098044395447, 1.018036127090454, -1.7477856874465942, 0.3273864686489105, 1.0268990993499756, 0.9262863397598267, -0.19544482231140137, 0.4554249346256256, -0.34126901626586914, -1.6564929485321045, -0.020174361765384674, -1.2140148878097534, 0.7470558881759644, -0.7028835415840149, 1.8893024921417236, -1.2678474187850952, -1.918447732925415, 0.10429048538208008, -1.34916090965271, 0.5120405554771423, -0.4886321425437927, 0.48697206377983093, -0.4047168791294098, -0.2638929486274719, 0.35743317008018494, -0.3553614616394043, 0.3898380398750305, -1.6774929761886597, -0.27774113416671753, 2.089323043823242, 3.01046085357666, 1.7325299978256226, -0.6741759777069092, 0.7400872707366943, 0.7328580021858215, -0.5335443019866943, -0.8260652422904968, -1.27151358127594, -0.9333855509757996, -0.6411970853805542, 0.2253759801387787, 0.7783210873603821, 0.31560975313186646, 0.06852748245000839, -0.5652256608009338, 0.4565787613391876, -0.9492173194885254, -1.4636667966842651, -1.0680452585220337, -1.6290769577026367, -0.20989923179149628, 0.2299347072839737, 0.584086537361145, -0.9386611580848694, 2.466700792312622, 0.4902009963989258, 1.3741388320922852, -0.13042724132537842, 0.4835336208343506, -0.3260371685028076, -0.43071842193603516, -0.18455757200717926, 0.22430847585201263, 1.4259655475616455, -0.05797345191240311, 1.7671825885772705, 0.12266324460506439, 0.0075119794346392155, 0.8732424378395081, 0.05042356997728348, -1.330010175704956, 0.20363251864910126, 0.3135520815849304, -0.8626662492752075, -1.1993488073349, 1.7838155031204224, 0.7645031809806824, 1.9385108947753906, -0.15809334814548492, 1.9068185091018677, -0.31031620502471924, -1.8166296482086182, -1.5889250040054321, 1.5728099346160889, -0.9636968374252319, -1.069871425628662, -0.4866698384284973, 0.9282272458076477, -1.0055538415908813, -0.2736036479473114, -0.4003250300884247, 0.011387371458113194, -1.6698095798492432, 0.15349406003952026, -1.2810920476913452, -0.9603993892669678, 1.3723106384277344, -0.1364683359861374, -2.367676019668579, -0.18931537866592407, -0.35565391182899475, -0.14191703498363495, -0.6721847057342529, -0.890005886554718, 0.07018603384494781, -0.06901853531599045, -0.719920814037323, -1.1853468418121338, 0.4151982069015503, -0.45389845967292786, 0.7156105637550354, -1.0555520057678223, -0.8377936482429504, 0.6204683780670166, -0.8453202247619629, -0.10212689638137817, 0.7728626728057861, -0.3361132740974426, -0.1143624410033226, -1.265325665473938, 0.29689422249794006, 0.8206124305725098, 0.21408669650554657, 1.7115143537521362, -0.06236867606639862, 0.8275309801101685, -0.5019171237945557, -0.3942002058029175, 1.7002179622650146, 0.357887327671051, 1.540031909942627, -1.2060260772705078, -1.3290430307388306, 0.30657848715782166, -0.5907264947891235, -0.2239723652601242, -0.7926075458526611, 0.1619139313697815, 0.333332896232605, -0.1659778207540512, 2.3003013134002686, -0.88548743724823, 0.6651337146759033, 0.9681976437568665, 1.7356350421905518, -2.5141518115997314, 0.15125463902950287, -0.27017149329185486, -1.1774237155914307, -0.7482132911682129, -0.7377604246139526, 1.1102092266082764, 1.999262809753418, 0.014960241504013538, 0.11538109928369522, 1.1498479843139648, 1.463085651397705, 1.051817774772644, -0.9977497458457947, 0.03283795714378357, 1.4253771305084229, -1.2067360877990723, 0.02220870740711689, -0.20942774415016174, 0.008302463218569756, -0.49217525124549866, -1.3295401334762573, 0.7432507276535034, -0.055428341031074524, -0.7726231217384338, -0.6661116480827332, -0.12880758941173553, -2.247313976287842, 1.3348801136016846, 0.9123289585113525, 0.4353311061859131, -0.6450116038322449, -1.2416415214538574, 2.6185760498046875, 0.11009228229522705, -0.23128294944763184, 1.8782193660736084, -0.5032833218574524, 0.9376694560050964, -1.2927464246749878, -1.3874232769012451, -1.7760916948318481, 0.23580506443977356, 0.5845791697502136, 0.12090510129928589, 0.19576382637023926, 1.0753631591796875, 0.09808965027332306, -0.7654810547828674, 0.37600991129875183, -0.741673469543457, 2.03475022315979, -0.8446844220161438, 1.2519525289535522, 0.6335713267326355, -1.2951867580413818, 1.381568193435669, 0.7832409739494324, 1.1124187707901, -0.986150860786438, -0.43666738271713257, -0.032457541674375534, 0.41667959094047546, -0.3142830729484558, -0.13189418613910675, -2.0519204139709473, 0.6086854934692383, -0.4123346209526062, 1.6546355485916138, 1.6152358055114746, 1.1443989276885986, -0.13083171844482422, -1.9801383018493652, 0.2442130446434021, 0.5067396759986877, 1.057828426361084, -0.839270293712616, -2.0024635791778564, -0.31322160363197327, -0.14178264141082764, -0.10574807971715927, -0.13532094657421112, 0.9170718789100647, -0.6254073977470398, -1.238312005996704, -0.48744410276412964, 1.3153828382492065, 1.3702857494354248, -0.053006138652563095, 1.1685452461242676, 2.1532201766967773, 0.1740640550851822, 0.5081782937049866, 0.607257604598999, -0.6604056358337402, -0.1475464403629303, 0.00814431719481945, 0.46229416131973267, -0.9317179918289185, 0.720844566822052, 0.09445939213037491, 1.3805055618286133, -0.9340825080871582, -0.0181554164737463, -0.15292486548423767, 0.10858739167451859, -0.7928085923194885, 1.7587448358535767, -1.2710206508636475, 0.798991858959198, 0.7176530957221985, -1.4003556966781616, -0.13545116782188416, 0.44268906116485596, 1.1890079975128174, -0.9704640507698059, -0.45291364192962646, 2.0193920135498047, 0.7370357513427734, 1.4074819087982178, 2.0508933067321777, -0.3049745559692383, -1.6042667627334595, 1.3965916633605957, -0.36339378356933594, 0.21700426936149597, -0.8075446486473083, 0.5515861511230469, 0.10431144386529922, -0.413293719291687, -0.9217705130577087, 0.40707892179489136, 0.8641396164894104, -0.9384562969207764, 1.0453264713287354, 1.1107892990112305, -1.783648133277893, -0.09898069500923157, 0.794049859046936, 0.3829650580883026, -0.13931089639663696, -0.5282509326934814, 0.12381142377853394, 0.13782377541065216, -0.2423006147146225, -0.19790968298912048, -0.05132526159286499, 0.49908187985420227, -1.3882975578308105, -1.692427396774292, -0.5714029669761658, -2.117691993713379, 0.17611220479011536, 1.0209732055664062, -0.3510524034500122, -1.2890596389770508, -1.7304531335830688, -0.812927782535553, -0.6639413237571716, -0.3417429029941559, -0.3199867010116577, -0.3951396644115448, 0.17241571843624115, 0.2076999545097351, 0.8331297039985657, 1.3367094993591309, 0.4617210030555725, -0.9365255832672119, 2.2860822677612305, 0.7407549619674683, -0.644608199596405, -0.4620504677295685, 0.03763488680124283, 0.8092900514602661, -0.18058858811855316, 1.1265778541564941, -0.4029216766357422, 0.9176721572875977, -0.7886549830436707, 0.7072068452835083, -1.5580949783325195, -0.19844168424606323, 0.6743882894515991, 0.8476694226264954, -1.745545744895935, -2.024843454360962, 0.4234775900840759, 0.7378765344619751, 1.5668466091156006, -1.4563332796096802, 0.8899574875831604, -0.25469261407852173, -1.5240031480789185, -0.455651193857193, 0.12544894218444824, -0.28641659021377563, 0.32338476181030273, -0.05794905498623848, -2.793898344039917, -1.4938790798187256, 0.13303668797016144, 0.04592578485608101, 0.8971273899078369, 0.24353240430355072, -0.5955632328987122, -1.0423345565795898, -0.6371053457260132, -0.25503507256507874, 0.2740892767906189, 0.11412075906991959, -0.16069142520427704, -0.4913072884082794, 0.7915984988212585, -0.1567775309085846, 1.6858594417572021, 1.0883439779281616, 2.045542001724243, -0.1413751244544983, 1.0902549028396606, -1.2263704538345337, -1.2125816345214844, -0.4091687798500061, -0.1142609715461731, 1.0944753885269165, 2.420534372329712, -1.8232343196868896, -0.6351487636566162, -0.9465770125389099, -1.0749198198318481, 0.9505958557128906, -0.4432367980480194, -0.44007304310798645, 0.37468352913856506, -0.3705582916736603, 0.443509578704834, -0.01577567495405674, 0.18603914976119995, 0.5913456678390503, 1.687995195388794, 1.2389411926269531, -1.0940937995910645, 0.9058927297592163, -1.308509111404419, -1.3968883752822876, 0.8741016983985901, 0.8610697388648987, 0.10538861155509949, 0.8699228763580322, -0.838701069355011, -0.007314283400774002, -0.6596779227256775, -1.2052384614944458, -0.15581324696540833, 0.47040891647338867, -1.3382360935211182, 0.11763180047273636, 0.3583467900753021, 0.1847415566444397, 0.7761963605880737, -0.7432385683059692, 0.023620517924427986, 0.5546333193778992, 0.4248618185520172, -1.2467907667160034, -0.8264517188072205, -1.451870083808899], [1.0084093809127808, -0.8644049167633057, 0.6209363341331482, 0.4282633364200592, 0.07913389056921005, 0.1145382672548294, 1.1933130025863647, -0.36304929852485657, -0.07175032794475555, 0.030238263309001923, 1.3742613792419434, 0.12957431375980377, -1.311745285987854, -0.03441178426146507, -0.8371915817260742, 0.039200276136398315, 0.8658791780471802, -1.8577139377593994, -0.5554517507553101, -0.7995668053627014, -0.2191181480884552, 1.2310981750488281, 0.9848439693450928, -0.9576236009597778, -0.873973548412323, -0.6795287132263184, 1.2610539197921753, 1.1713120937347412, 1.683556079864502, 0.7676793932914734, 0.9027124047279358, 1.1681993007659912, -0.49795252084732056, -0.17081022262573242, -0.0792534202337265, -0.5702829360961914, -0.8072507977485657, -0.4294024705886841, 0.7727571725845337, -0.46899235248565674, -1.735746145248413, -0.8109987378120422, 1.7422655820846558, 0.3290499448776245, 0.21241635084152222, 1.169465184211731, 1.0132302045822144, -0.29681628942489624, 1.187023639678955, -1.1945549249649048, -0.11048233509063721, 0.12430732697248459, -0.7042582035064697, -0.3171458840370178, -0.04108361527323723, -2.1075286865234375, 1.6857749223709106, 1.6750198602676392, -0.19249728322029114, -0.5891498327255249, 0.43128737807273865, 1.5087288618087769, 0.9767901301383972, -0.43588241934776306, 1.4142857789993286, -1.603981852531433, 1.5470693111419678, 0.6997631192207336, 0.2734391689300537, -0.3413408100605011, -1.3209682703018188, 0.8080760836601257, 0.40340274572372437, -1.0262120962142944, -0.5864109992980957, 0.15781253576278687, -1.727826476097107, -0.11705728620290756, -0.6446756720542908, 1.1193811893463135, -1.091556191444397, -1.4646453857421875, 1.402859091758728, 0.9839804172515869, 1.6078463792800903, 1.7543467283248901, 0.88652503490448, 0.1396055370569229, -0.42190420627593994, 1.1634100675582886, -1.2847967147827148, 1.1287235021591187, -0.1927889883518219, 0.08660081028938293, -0.9662380218505859, 0.38845568895339966, -0.06006813794374466, -0.3643740713596344, -1.8506547212600708, 0.8567308187484741, -0.5755185484886169, -0.3950391709804535, 0.06640598922967911, 1.0683979988098145, 1.4015744924545288, 0.5999293327331543, 0.4827441871166229, -0.25273004174232483, -0.060515329241752625, -1.7216832637786865, 1.152942419052124, -0.5800198316574097, -1.0924859046936035, -1.3736572265625, 1.2106729745864868, 0.48178234696388245, -0.35516557097435, 0.9786068797111511, -0.28883200883865356, -0.5863690376281738, -0.19403783977031708, -1.4147379398345947, -1.4925209283828735, 0.8610363602638245, 1.0408482551574707, 1.7302254438400269, 1.107832670211792, 0.9696766138076782, 0.882532000541687, 0.2633247673511505, 2.0894808769226074, 0.6546575427055359, 1.195108413696289, -0.9147436022758484, -0.11353475600481033, -0.5963572859764099, 0.027800295501947403, 0.9400990009307861, 1.0318598747253418, -1.084558129310608, -0.8724017143249512, 1.086184024810791, 0.01914278417825699, 0.8261121511459351, -1.51710844039917, 0.913359522819519, 1.0924720764160156, 0.010512711480259895, -0.23048318922519684, -0.954312801361084, 1.0731598138809204, -0.19285990297794342, 0.28639763593673706, 1.2112642526626587, 0.4999014437198639, 1.4201322793960571, 0.6762259602546692, -0.09357679635286331, 0.7823204398155212, -2.067120313644409, 0.844227135181427, -0.08128854632377625, -1.1204888820648193, -0.28255534172058105, 1.9562079906463623, 0.20040437579154968, -0.058242857456207275, 0.5879780650138855, -0.4900647699832916, 0.07757211476564407, 0.4678782820701599, -1.4157977104187012, 1.0794919729232788, 0.12608981132507324, 0.26118093729019165, 2.518859386444092, -1.1333658695220947, 0.712955117225647, -0.9021412134170532, -0.01312093436717987, -0.345859557390213, 1.841666340827942, 0.46588966250419617, -0.6855356097221375, 1.3726834058761597, 0.2467688024044037, 2.3060803413391113, 0.04906342923641205, 0.06853656470775604, 0.5763769745826721, 0.5685081481933594, 0.864528238773346, 1.4394879341125488, -1.3905783891677856, 0.513824462890625, -0.14767634868621826, 0.39691171050071716, -1.2216269969940186, 1.8145437240600586, -2.27653431892395, 0.43990302085876465, 1.3318700790405273, 0.17028774321079254, -0.7733973264694214, 1.323383092880249, 0.9823853373527527, -0.054274845868349075, -0.46321186423301697, 1.3304141759872437, -1.1391572952270508, -1.0654592514038086, 0.8536874651908875, -0.1917392760515213, -0.18849343061447144, 2.0788753032684326, -1.0725383758544922, -2.06459903717041, -0.24526356160640717, -0.7791774272918701, 0.6023486256599426, 3.107783794403076, -1.0740653276443481, -0.04974035173654556, 0.8068488240242004, 0.4902224540710449, -0.9441048502922058, 0.05773448944091797, -0.23419184982776642, -0.5043299198150635, 0.10323993861675262, -0.3274412155151367, 0.35700011253356934, -0.15578322112560272, -1.3838046789169312, -0.5451629757881165, -0.3743251860141754, 0.8163660764694214, -0.2402266561985016, -0.6904807090759277, -1.007139801979065, -0.2919868230819702, -0.612404465675354, 0.49611613154411316, -0.34848448634147644, 0.17660300433635712, -0.5763411521911621, -0.03506306931376457, -0.19644412398338318, -0.1172729954123497, -0.9111077189445496, 1.7269279956817627, 0.21237413585186005, 0.7464522123336792, -2.359962224960327, -0.34315240383148193, -2.0285677909851074, -0.23736310005187988, 1.8448501825332642, 0.5727092027664185, -0.9747412800788879, 0.013703139498829842, -0.28095000982284546, -0.010641781613230705, -0.12025271356105804, -0.5640828609466553, -0.0761793926358223, 0.07332754880189896, -0.404519259929657, 0.5230896472930908, 1.0140271186828613, -0.7089340090751648, 0.6065343022346497, -0.016876477748155594, 2.5193862915039062, 1.0610418319702148, -0.312694251537323, 0.5077597498893738, -0.5022372603416443, -1.062729835510254, -0.4075261354446411, 0.1825120896100998, -0.6532937288284302, -0.8231952786445618, 1.298553705215454, -2.5895204544067383, 0.05876101180911064, -0.5144893527030945, -0.3536340594291687, -0.12693603336811066, 0.2730085849761963, 0.43252357840538025, -1.2355045080184937, -1.9722431898117065, -0.27246174216270447, 0.11485602706670761, 1.0584059953689575, -0.9161413908004761, 0.0021713310852646828, 0.3654513359069824, -0.8701576590538025, -0.9416553974151611, -0.7513465285301208, -0.3337550759315491, -0.13325347006320953, -1.2691127061843872, 1.1220914125442505, 2.1846020221710205, 0.3504582345485687, -0.3041995167732239, 1.0584524869918823, -1.589493751525879, 0.5140326619148254, 1.3087365627288818, 0.5061746835708618, -1.1388823986053467, -0.5726355314254761, 0.5636438727378845, 0.7901911735534668, 1.1867170333862305, 1.443634033203125, -0.12991870939731598, 0.5672705769538879, -0.5685860514640808, -0.33256179094314575, -0.07182648032903671, 0.3777908384799957, 0.9852798581123352, -0.38060063123703003, 0.8825406432151794, -0.33176058530807495, 0.7496548891067505, -1.002327799797058, -1.9416359663009644, -0.5709546804428101, -0.4273354113101959, 1.055516004562378, 0.21058285236358643, -0.8405593633651733, -0.6011897325515747, 0.42723068594932556, 0.2238156795501709, -0.5662238597869873, 0.8927026391029358, -0.5627083778381348, -0.23112893104553223, -0.668354332447052, -0.23861946165561676, -0.44591692090034485, -0.6869527697563171, -0.32977503538131714, -0.3761826157569885, 0.5681428909301758, -0.47361841797828674, -0.5806041359901428, -2.0095174312591553, 2.7320468425750732, -1.895460844039917, 0.16896802186965942, -0.4958837330341339, -0.4800437092781067, 1.0140495300292969, -0.20135602355003357, 0.44846442341804504, 0.1352730095386505, 0.4022878408432007, 1.9428846836090088, -0.2301350086927414, -0.2902214229106903, -1.2864652872085571, -2.0686309337615967, 0.49771153926849365, 1.091291069984436, -0.5989727973937988, -1.3335484266281128, -0.9884699583053589, 1.2305105924606323, 0.6207820177078247, 0.20288309454917908, 1.0087754726409912, 1.7640259265899658, -2.4899771213531494, 0.6998181939125061, -0.16874973475933075, -0.5339862108230591, 1.3156038522720337, 1.0676639080047607, 1.0735667943954468, 1.3700876235961914, -0.13347958028316498, -1.4040359258651733, 1.1529630422592163, 1.266303300857544, 1.518943428993225, 1.2712321281433105, 0.9052395224571228, 0.10928903520107269, 0.7777360677719116, 0.7528443932533264, -0.8903780579566956, 0.5255033373832703, 1.610079050064087, 0.1506221741437912, -1.2409477233886719, 1.3057390451431274, 1.5454567670822144, 0.01164236105978489, -1.9846757650375366, 1.5088106393814087, -0.7510050535202026, 1.6186904907226562, 0.09681206196546555, -2.24721360206604, 0.20848287642002106, -0.2039259821176529, -0.6347165703773499, 2.201878309249878, -1.5091544389724731, 0.6674294471740723, -0.12041456997394562, 0.25928637385368347, -0.7843758463859558, -0.6900697350502014, 1.0904585123062134, 1.05764639377594, 0.5699771642684937, -0.39152100682258606, -1.6728371381759644, -0.43190568685531616, 0.9345526099205017, -1.3783884048461914, 0.6585928201675415, -0.4809578061103821, 0.0926424190402031, -1.1693516969680786, 0.3869040608406067, -2.7369213104248047, -0.11660310626029968, -1.2038264274597168, -1.4035638570785522, 0.6938037872314453, -1.5259524583816528, -0.503929615020752, -0.3424050509929657, -0.08043396472930908, 0.005092478822916746, 0.3732878267765045, 2.6069676876068115, 0.9559135437011719, -0.027830801904201508, 1.2623798847198486, -0.4030139446258545, -1.0628714561462402, 1.1036992073059082, -1.1846446990966797, 1.4880635738372803, -0.1503058671951294, -0.44058990478515625, -0.10342293232679367, 1.7770891189575195, -0.662487804889679, -0.725542426109314, 0.9350367188453674, -0.7621080875396729, 0.34406936168670654, -0.9539306163787842, 0.823951244354248, -0.5337828397750854, -1.1555103063583374, 1.9812414646148682, 1.2218571901321411, -0.41009974479675293, 0.08871171623468399, -0.584248423576355, -0.7452601194381714, -1.0512902736663818, 1.7589102983474731, -0.6071892380714417, 0.09567611664533615, -0.42124003171920776, 1.0548003911972046, 1.5171114206314087, 0.6965346932411194, -0.4429984390735626, 0.2509828805923462, -1.9018168449401855, 1.1197854280471802, 1.073634386062622, -0.16226889193058014, 0.9459161758422852, -1.3119103908538818, -0.4186396300792694, 1.782127022743225, -1.2979168891906738, 0.5270909070968628, 0.3966837525367737, -0.1695953756570816, -0.459185928106308, 0.2406337410211563, 0.16401252150535583, -0.9874309301376343, 1.1453816890716553, 0.734766960144043, 0.6104494333267212, 2.082494020462036, 1.1813937425613403, -0.07560397684574127, -0.8817757964134216, -3.1466598510742188, -0.8858622908592224, 1.0289815664291382, -0.9871195554733276, -1.1186972856521606, -0.1662103682756424, -1.2000412940979004, -0.16166086494922638, -1.2760142087936401, -0.9789595007896423, -0.6710243225097656, 0.6636791825294495, 0.411289244890213, -0.09029223769903183, 0.12057869136333466, -0.49746784567832947, -0.12469532340765, -1.0656355619430542, -0.28221458196640015, 0.7636414170265198, 0.45799294114112854, 0.29260554909706116, 0.3194305896759033, 0.947566568851471, 1.1908957958221436, -1.4559364318847656, 0.9206664562225342, -0.3453752398490906, -0.5347960591316223, -1.271444320678711, 1.7648441791534424, -0.8379240036010742, -1.5564674139022827, -0.38589853048324585, -1.1540111303329468, -1.2141163349151611, 0.7464174032211304, -0.5339120626449585, -1.0320041179656982, -0.860295295715332, -0.010480405762791634, -0.5422960519790649, 2.4298644065856934, -0.11258145421743393, -0.2687399685382843, -1.4565984010696411, 2.0773751735687256, -1.6708050966262817, -0.4724671244621277, -0.9418349266052246, 1.671130895614624, 1.165168285369873, -0.6054382920265198, 1.3081530332565308, 0.23123544454574585, -0.518902063369751, 1.4333726167678833, 0.05821870639920235, -0.62473064661026, 0.7527378797531128, 1.4364521503448486, -0.18063728511333466, 0.18839940428733826, 1.3971621990203857, -0.5835351347923279, -0.1869777888059616, -1.2968380451202393, -0.20759204030036926, -1.0680851936340332, -0.2993607223033905, -0.7210599184036255, 0.3355618417263031, -0.6099351644515991, 1.59575355052948, -0.08912382274866104, -0.8969129323959351, 0.014807152561843395, 0.3262758255004883, 1.1439039707183838, -0.6136911511421204, -0.8461293578147888, 0.10197614133358002, 0.7837309837341309, 0.7672555446624756, -2.1253840923309326, -1.1342546939849854, 0.36561015248298645, 0.1604306548833847, -2.56056547164917, -0.056981898844242096, -0.8989623188972473, 0.26137545704841614, 0.19691334664821625, 0.5220058560371399, -1.7225233316421509, -0.4561885595321655, -0.2664068937301636, -0.24733103811740875, -0.9156325459480286, -1.7000316381454468, -0.6991307139396667, -1.1000937223434448, -0.38814395666122437, 1.6932766437530518, 0.07204420119524002, -1.3634966611862183, -0.5598084330558777, -0.1916348785161972, 0.6584640145301819, -2.2638556957244873, 1.3245530128479004, -0.9030329585075378, -1.6532491445541382, 1.0284614562988281, -0.7597441077232361, -0.46362820267677307, -0.6478586792945862, 0.9978489875793457, 1.1269285678863525, -0.6982783079147339, -0.4782845973968506, 0.4388904273509979, -0.5935690999031067, 0.0875258520245552, 0.21667088568210602, -0.5177256464958191, -0.2438361793756485, 0.029949193820357323, -0.6790105104446411, 2.191842794418335, -1.251941442489624, 0.9860514402389526, 0.3243027925491333, 0.18755698204040527, -2.62406325340271, -0.8528687357902527, -0.04782133176922798, 0.774192750453949, -0.0185908954590559, 0.7008641362190247, -0.9118286967277527, -1.0143035650253296, 0.5756906270980835, 0.49544215202331543, -1.6127263307571411, 0.5909994840621948, -1.1297663450241089, -0.2453281432390213, 0.8604313731193542, -0.1497000753879547, 2.436599016189575, 1.5552908182144165, 0.736656665802002, -2.202146053314209, -0.4107207953929901, -0.5410231947898865, -0.22496727108955383, -0.2131582349538803, -1.788259506225586, 0.39242851734161377, -0.9396747350692749, -0.3589814007282257, 0.36776211857795715, 1.4122905731201172, 0.7162569165229797, -1.2820333242416382, 0.6469306349754333, 0.4665064215660095, -0.12645873427391052, 1.6139189004898071, -0.38792577385902405, 0.3476412296295166, 2.2179946899414062, -0.5633615851402283, -0.7440720796585083, 0.6617629528045654, 0.5196205973625183, 0.10005571693181992, -0.11638688296079636, 0.19965289533138275, 0.7700883150100708, -1.5564850568771362, 0.01765761710703373, 1.0768712759017944, -1.016036033630371, 1.116678237915039, 0.047591693699359894, 0.07614673674106598, -0.31156569719314575, 0.9570826292037964, -0.33231475949287415, 1.0664643049240112, -0.0008378541097044945, -0.11842000484466553, 1.4740031957626343, -1.5257118940353394, -0.4282357692718506, 0.35725417733192444, 1.319219946861267, -0.6006031036376953, -0.12284963577985764, -0.6059670448303223, 0.23678524792194366, -0.262746661901474, -1.246826171875, 1.1695795059204102, 0.05385966598987579, 0.014546552672982216, -0.506261944770813, 1.2441356182098389, -0.24092641472816467, 0.4608844518661499, 0.6067044138908386, 1.7701457738876343, 0.7296796441078186, -0.2808989882469177, -0.270981103181839, 0.3926229476928711, -1.215714693069458, 0.1744968742132187, 1.0081275701522827, 1.5194822549819946, 0.17232617735862732, 0.8211826086044312, 0.4051341116428375, -0.011022062040865421, -1.776801347732544, -0.3141663074493408, 0.9622313380241394, 0.07745322585105896, 1.172461748123169, -0.7530659437179565, 1.5240888595581055, -1.9178162813186646, -0.5176776051521301, -1.9229907989501953, -0.002468975493684411, -0.24473528563976288, 0.5576306581497192, 1.3157676458358765, 0.06086180731654167, 0.18811377882957458, -1.2256499528884888, 0.12379588931798935, -0.6472005844116211, -0.8264755606651306, -0.39033910632133484, 0.37530672550201416, 0.9636746048927307, -0.3431013822555542, -0.223097562789917, 0.7137653827667236, 0.6503947377204895, 0.5620886087417603, 0.4386025369167328, -0.7352032661437988, -0.4363519251346588, -1.108488917350769, 1.2213160991668701, 1.3290592432022095, -1.8124815225601196, 0.4415249228477478, 0.6406148076057434, -0.5844061970710754, -0.5133653879165649, -0.18134260177612305, 0.4237145483493805, 2.6910715103149414, -0.047748394310474396, 0.8732319474220276, -1.0782917737960815, -0.3211009204387665, -1.6105316877365112, 1.1395436525344849, -0.2618025243282318, 0.08036229014396667, -1.5690066814422607, 1.9989314079284668, 0.30566489696502686, -1.5769333839416504, -0.11856072396039963, -1.2162057161331177, -0.21589191257953644, -0.8434592485427856, 1.0307356119155884, -1.4058009386062622, -0.5928440690040588, 0.520419180393219, 0.8039630055427551, -0.7299857139587402, -0.9572217464447021, -1.982008695602417, -1.6003128290176392, -1.3278263807296753, 0.4680219292640686, -1.7258135080337524, -1.0082404613494873, -0.014185260981321335, 0.5836926102638245, 0.16168320178985596, 0.844058096408844, -0.14939925074577332, -0.20324909687042236, -1.0786627531051636, -0.9309016466140747, -0.05427689850330353, -1.2953743934631348, -0.35218825936317444, 0.9631378054618835, 0.9175106883049011, -0.5476603507995605, -1.1384398937225342, -0.18927301466464996, 1.3832950592041016, 1.9208158254623413, -2.2480506896972656, -1.6174275875091553, 0.6361578702926636, 0.8953825235366821, 0.8245815634727478, 0.21398064494132996, 1.8340368270874023, -1.157168984413147, -1.7369612455368042, -0.027750683948397636, -1.124093770980835, 0.19305051863193512, -0.44359588623046875, 1.306749939918518, 1.4711357355117798, 1.5404287576675415, -0.37925490736961365, 0.4585639238357544, 0.6676915287971497, 0.5389013886451721, 0.06528017669916153, -0.3478536903858185, -0.5631411075592041, -1.3879187107086182, 0.7161977291107178, 0.2106562852859497, -0.8118163347244263, -0.11516343802213669, -2.1463098526000977, 1.552724003791809, 0.40055716037750244, -0.49238649010658264, 0.20636701583862305, -1.6110725402832031, 1.6659443378448486, 0.19110015034675598, -0.052433181554079056, -0.007781213615089655, 0.15386290848255157, -0.7547163367271423, 1.6354819536209106, -0.06567654758691788, -0.3610784113407135, -0.49826768040657043, 0.16343295574188232, -0.6577292680740356, -0.19707919657230377, -1.9408637285232544, -0.06280718743801117, -0.6550964117050171, 1.2965925931930542, 0.2934047281742096, -1.4543368816375732, 0.668190598487854, 1.0215225219726562, -0.5874514579772949, 0.3261151909828186, 0.015007497742772102, 1.4877780675888062, 0.788846492767334, -0.07596994936466217, -1.168887972831726, -0.6502446532249451, 1.1840548515319824, 0.7842321991920471, -2.1922783851623535, -1.9957058429718018, 0.5814277529716492, -0.43778547644615173, -0.6248124241828918, 0.3516988158226013, -0.4291982352733612, -0.8997917175292969, 1.071770429611206, 1.7061861753463745, 0.31433698534965515, 0.034751418977975845, 1.2400672435760498, 1.7263201475143433, 0.0810059905052185, 0.6811189651489258, 0.0011371758300811052, 3.344388008117676, -1.3332659006118774, 0.031535662710666656, 0.15179793536663055, 0.005437117535620928, -0.12955552339553833, -1.7614359855651855, -0.5818546414375305, -0.3004626929759979, 1.64681875705719, -1.5276379585266113, -0.4239612817764282, 2.5664923191070557, 0.42612388730049133, 0.22923396527767181, 1.8603442907333374, 0.21620315313339233, 0.06621083617210388, 0.6071948409080505, -1.5294071435928345, -0.011058350093662739, -0.9015923738479614, -1.248058795928955, -0.6142850518226624, -0.3311147093772888, -1.016517162322998, 0.2871362268924713, -1.296924352645874, 2.3812410831451416, 0.4074687957763672, 1.1020818948745728, -0.12933598458766937, -0.5079724788665771, -0.3449902832508087, -0.4045672118663788, -0.28741195797920227, -1.0007603168487549, 0.25883403420448303, 1.4148401021957397, 0.4808637499809265, 0.2479233592748642, 0.4350014626979828, -1.7150620222091675, 1.5571359395980835, 1.3179504871368408, 0.34214940667152405, -1.0506510734558105, 2.300102472305298, 0.26571646332740784, -1.6198740005493164, 1.2816413640975952, -1.0848976373672485, -0.6614949107170105, 1.3528193235397339, 0.6658744215965271, 1.1343703269958496, 0.06273481994867325, -0.6700582504272461, 1.936523675918579, -0.019374577328562737, -0.16685548424720764, 0.5134979486465454, -0.30303558707237244, 0.42717206478118896, -0.9054185748100281, -0.24917329847812653, -1.1516404151916504, -1.380237102508545, 1.0663114786148071, 1.3079726696014404, -1.0684562921524048, -0.5496293902397156, 0.3630337417125702, 0.853280246257782, 0.5380480289459229, 0.8273037075996399, -0.525283694267273, -1.7699366807937622, 1.624145746231079, 0.1813424676656723, 3.046755313873291, 0.5796144604682922, 0.41450151801109314, -0.553464412689209, -1.7484891414642334, 0.17373061180114746, -1.1558681726455688, -2.0499043464660645, -0.7482508420944214, 0.8413780927658081, -0.45720115303993225, 0.7028550505638123, -1.6765708923339844, 1.0677447319030762, 0.7062992453575134, -1.0135787725448608], [-0.8064015507698059, 0.017229586839675903, 0.3688495457172394, 0.1742350161075592, -0.028306743130087852, -0.13243047893047333, 0.5343894362449646, 0.5309028625488281, 0.3799341917037964, -1.2253541946411133, -0.23956215381622314, -0.5478346943855286, 0.794349193572998, -0.6767521500587463, -0.5989671945571899, -0.5634809732437134, -0.4106055498123169, -1.7575582265853882, 0.004828039091080427, -0.7015771269798279, 0.1571141481399536, 0.9323076605796814, 1.0924698114395142, -1.2178367376327515, -0.1880410760641098, -1.0052355527877808, 1.1908066272735596, -0.7929452657699585, -0.6814943552017212, 0.10062377899885178, 1.0006498098373413, -0.545518159866333, -1.7627440690994263, -0.24540852010250092, -0.662222146987915, -1.130203366279602, -0.3943079710006714, -0.13446566462516785, -0.4559563100337982, -1.0863538980484009, -0.9784676432609558, -0.7018921971321106, -0.9066882133483887, -0.8903847932815552, 2.4535906314849854, -0.3453730642795563, -0.21886134147644043, 1.3404536247253418, -1.567399024963379, -0.6476058959960938, -2.3738791942596436, 1.5224395990371704, 0.9601825475692749, -0.6835514307022095, -1.5720824003219604, -1.042453646659851, 0.7008545398712158, -0.14991340041160583, -1.2133253812789917, -0.1883023977279663, 0.47110387682914734, -0.4625089168548584, 0.5738282799720764, -1.6848902702331543, -0.055467721074819565, 0.9301708340644836, 0.8795902132987976, 0.4483359158039093, -0.33449211716651917, -1.5253719091415405, 2.1827220916748047, -1.0960532426834106, -0.3564727008342743, 2.093109607696533, -0.1444970816373825, -1.3983060121536255, 0.024050015956163406, -0.07426247000694275, -0.1144498661160469, -1.2108012437820435, 1.5561842918395996, 0.1559082716703415, 0.07479938864707947, -1.22428297996521, 1.0254606008529663, 2.3202786445617676, -0.3033042848110199, -1.0776646137237549, 0.35173559188842773, -2.021010160446167, 0.2591525912284851, -1.4896081686019897, -0.12004587799310684, 1.933388590812683, -0.6535043716430664, -2.1790053844451904, -0.5469116568565369, 0.7717640995979309, -1.2765153646469116, 0.6561295390129089, -0.22698941826820374, 0.039231158792972565, -1.3527864217758179, -0.36793065071105957, -0.6834731698036194, -0.01069632638245821, -0.8546491265296936, -1.3505593538284302, -2.1301329135894775, -0.31335821747779846, -0.526177704334259, 0.5824574828147888, -0.37669074535369873, -1.2081959247589111, 0.5385795831680298, -1.7501685619354248, -1.20253324508667, -0.19278135895729065, -0.43067988753318787, -0.6090670824050903, -0.11799473315477371, 1.0391197204589844, 1.3575583696365356, -1.0063668489456177, -1.1320756673812866, 1.3884273767471313, -0.4066166877746582, 0.27829092741012573, -1.147624135017395, 0.6435712575912476, 1.6621463298797607, 0.8378666639328003, -0.9726304411888123, -0.011624407023191452, 0.46135255694389343, -0.7714359164237976, 0.1471574902534485, -1.7545037269592285, -1.0076630115509033, 1.0221365690231323, -0.5755795240402222, 0.45351988077163696, -0.07058651745319366, 0.5801573395729065, 0.4405611753463745, 0.3441402316093445, 0.22851000726222992, -0.3207760751247406, -0.992706298828125, -0.3704187273979187, 0.5139621496200562, -0.7615489959716797, -0.1208299919962883, 0.5086022019386292, -0.4070299565792084, 0.23149003088474274, -0.7281211614608765, -0.06739875674247742, 0.4206794798374176, -0.36677974462509155, 1.7184710502624512, 0.2753216028213501, -0.15969125926494598, 0.2870340347290039, 0.6516790986061096, -1.4571529626846313, 1.169602394104004, 0.16548509895801544, 1.4559463262557983, -0.0028213136829435825, -0.49178174138069153, 1.2055660486221313, -0.20520715415477753, 0.12850138545036316, -0.7254571914672852, -1.064810872077942, -0.28542274236679077, -1.2894688844680786, -0.3382706642150879, -0.6067303419113159, -0.14077548682689667, -0.8311799764633179, 0.43464797735214233, -0.3539678454399109, 3.7365529537200928, 1.2157316207885742, 0.9461154937744141, 0.9932973980903625, 1.8681622743606567, 2.2451171875, 0.13728247582912445, -0.25935375690460205, 0.20394866168498993, 1.8638538122177124, -0.16371652483940125, 0.27633729577064514, -0.4995567500591278, -1.0520782470703125, -0.48760274052619934, 0.7204124331474304, -0.0699893906712532, -1.7267529964447021, 0.3298541307449341, -0.8987631797790527, -1.086361289024353, -0.308270126581192, -2.8965060710906982, 1.375853419303894, -0.6037130951881409, -1.1432232856750488, 0.24301715195178986, 1.6098188161849976, 0.16390977799892426, 0.8184814453125, -1.3736025094985962, -0.1896800696849823, -0.09796332567930222, -0.9444130063056946, -1.165950059890747, -0.5886985659599304, 0.17903362214565277, 2.3118293285369873, -0.23341305553913116, -0.23619376122951508, -1.468105435371399, 0.8212952017784119, -0.33283737301826477, 1.9637447595596313, -1.103440523147583, 1.0334333181381226, 0.5412747859954834, -0.2856195569038391, 0.9225008487701416, -1.4528729915618896, -0.32379889488220215, 0.5572443604469299, 0.6838892698287964, -1.4385675191879272, 2.687242269515991, -0.9632949829101562, 0.9682213068008423, -1.0175975561141968, 1.4071842432022095, 0.7952014803886414, -0.05950813367962837, -0.3531583249568939, -0.8467565774917603, -1.112862229347229, 1.1163535118103027, 0.9745959043502808, -0.4764440059661865, -2.0271120071411133, 0.5990593433380127, 0.6737715601921082, -0.20700226724147797, 0.7377405166625977, -0.058727771043777466, 0.24714075028896332, -0.35541823506355286, 0.8678978085517883, 1.551814079284668, -0.6724025011062622, -1.0713915824890137, 1.4681740999221802, 1.4797050952911377, 0.7655118703842163, 0.20890618860721588, -2.2720398902893066, 2.0305073261260986, -1.2027153968811035, 0.2405005693435669, 0.2000628560781479, -2.0200910568237305, 0.5939080119132996, -0.7518969774246216, -0.1845633089542389, -0.7199134230613708, -0.6830205321311951, 0.1469334214925766, -0.3747645914554596, -0.7185173034667969, 0.039685215801000595, -0.5876916646957397, 1.4181854724884033, -0.2028535157442093, 0.023662641644477844, -1.046823501586914, 1.6661335229873657, 0.28673648834228516, -0.40326988697052, 0.9509480595588684, 0.8940321207046509, 0.5980756878852844, 0.5126367211341858, 1.5047781467437744, -0.8666093349456787, -0.2994614541530609, 0.26563388109207153, -0.5243228077888489, 1.063114047050476, -0.7430933117866516, 1.765916109085083, -1.7189221382141113, -0.15883013606071472, -0.48319822549819946, -0.3016045391559601, -0.25223061442375183, -0.49157771468162537, -1.1259714365005493, -1.5392550230026245, -0.14377306401729584, -0.8182833194732666, 0.5948428511619568, 1.035590410232544, 0.7544728517532349, -1.0098013877868652, 0.029994428157806396, -2.169816493988037, -0.8953929543495178, -0.8249944448471069, 0.6042911410331726, 0.7055587768554688, -0.0409061498939991, -1.4412424564361572, 1.190119743347168, -2.3065853118896484, 1.3927873373031616, 1.3691476583480835, -0.4240562319755554, -1.108775019645691, -0.47786185145378113, -2.038276433944702, 0.1767989695072174, 0.48518943786621094, 0.668807327747345, -0.4069374203681946, 0.5704875588417053, 0.7279192209243774, -1.1173574924468994, 1.3009315729141235, 1.3785898685455322, -0.13541264832019806, -2.465502977371216, 0.9972277283668518, 0.12151435762643814, 0.7274248600006104, -0.6664616465568542, 0.29693371057510376, -0.37806716561317444, -1.5576362609863281, 0.40157270431518555, 0.8615834712982178, -1.2312263250350952, 2.42618989944458, 0.7705395817756653, 1.2575478553771973, -1.587593674659729, -1.4083075523376465, -0.06424251198768616, -0.3795948028564453, -1.754629373550415, -0.33867624402046204, -0.19072137773036957, 0.548020601272583, -0.5875810384750366, 0.02746378444135189, 1.050307035446167, -0.07679152488708496, -0.9129804372787476, 0.9584649205207825, -0.15585821866989136, -0.09679211676120758, 1.4254542589187622, 0.0316004678606987, -0.37551766633987427, -0.03184415400028229, 0.39404061436653137, 0.9707001447677612, -0.8867321610450745, -2.203582286834717, 0.33376792073249817, -2.27298903465271, -0.5715442299842834, 0.39358100295066833, 1.0231274366378784, -0.26081469655036926, 0.46860411763191223, 0.07369931787252426, 0.38885626196861267, -0.7215287089347839, 0.577692985534668, -0.04572542384266853, -1.0716233253479004, 1.0251407623291016, -1.2538905143737793, 0.10678960382938385, -0.7154750227928162, 0.45096835494041443, 0.6070055365562439, 0.8951442837715149, -3.0059866905212402, 1.118963599205017, 0.17502470314502716, 0.4587636888027191, 0.9050478339195251, 0.8181913495063782, 0.9067307114601135, -0.19220934808254242, 1.316817283630371, -0.29953014850616455, 0.57820063829422, -1.034675121307373, 0.585539698600769, 0.2067999690771103, -1.2911285161972046, -0.37866953015327454, -0.5291301012039185, -0.5546607375144958, -1.1178758144378662, 1.924458384513855, 1.4934171438217163, -1.3802285194396973, 0.6666709184646606, -0.27376115322113037, -1.798061490058899, 0.4515683650970459, 1.5786356925964355, -0.25416114926338196, 0.2113751620054245, -0.4769642949104309, -0.071415476500988, 0.018445611000061035, -0.7896828651428223, 0.1303723156452179, -0.8383824825286865, 1.1136020421981812, 0.22666779160499573, 0.15028725564479828, -1.2228869199752808, -0.33228278160095215, -0.06735877692699432, -2.5992825031280518, -1.8401433229446411, 1.1259626150131226, 2.3409583568573, 0.39950937032699585, -0.3389744460582733, -1.4944816827774048, 0.4437982439994812, -0.2093687355518341, -2.0569159984588623, 0.2313779890537262, -1.3450218439102173, 0.7946959733963013, 0.233688086271286, -0.344667911529541, 0.6892362833023071, 0.48721352219581604, -0.005661176051944494, 1.4840826988220215, 0.272768497467041, -0.16012588143348694, -0.6348145604133606, 0.4405023157596588, -1.325445532798767, 0.3375944197177887, -0.6609233617782593, -0.7721601128578186, 0.13831818103790283, 1.1740853786468506, 1.7876594066619873, 2.357309341430664, -0.2540733814239502, 1.0459219217300415, -0.3328624367713928, -0.5443784594535828, 1.1949615478515625, 0.2697087228298187, 0.3565402030944824, -0.6806763410568237, 1.301712989807129, -0.6210419535636902, 0.042457424104213715, -0.34066617488861084, 0.12665419280529022, -1.0187618732452393, 1.2153831720352173, -0.1655302792787552, -0.8906795382499695, -1.0480815172195435, -2.624455213546753, -0.0936279296875, -0.5554296970367432, -0.8134810924530029, -0.9334595203399658, -0.6585236191749573, -2.5090816020965576, -0.04625615477561951, 0.39290058612823486, 0.2418978065252304, -1.3734997510910034, -0.4632795751094818, 0.7979562878608704, 0.5145686864852905, 1.8528645038604736, -0.055973030626773834, -0.40847721695899963, -0.8246744871139526, -0.1992262303829193, -2.0062060356140137, 0.06971905380487442, 0.34842416644096375, 0.4385640025138855, -0.6547000408172607, 0.5527017712593079, -0.23333419859409332, -2.015831708908081, 0.8311728835105896, -1.7128617763519287, 0.3168938159942627, -0.269853413105011, -1.6016438007354736, -0.5724551677703857, 0.22400963306427002, 1.1570823192596436, -0.7718369364738464, 0.15585055947303772, 1.3969889879226685, -0.8173588514328003, -0.3999935984611511, -1.0309330224990845, -0.981657087802887, -0.49410244822502136, 0.07942772656679153, 0.32312434911727905, -0.017330214381217957, -1.9460724592208862, -1.4377458095550537, 2.179955244064331, -1.0108877420425415, -0.9969037771224976, 0.3039849102497101, 0.6852980852127075, -0.342941552400589, 0.6172073483467102, -2.0148298740386963, 0.3218078017234802, 1.6122227907180786, 0.3358592391014099, -1.9253053665161133, 0.34769323468208313, 0.003918359521776438, -0.12894774973392487, 1.856748342514038, -1.8383821249008179, 0.5937923192977905, -1.6788893938064575, 2.18109130859375, 1.7699685096740723, 1.1618784666061401, -0.8504844903945923, -0.28312644362449646, 0.8677034378051758, 0.9815848469734192, 1.0132761001586914, 1.4406393766403198, -1.7303169965744019, 1.020591378211975, -0.7980299592018127, 0.6926171183586121, -0.5333484411239624, -0.3444004952907562, -1.352749228477478, -0.4949721097946167, -1.9798740148544312, -0.6048232913017273, -1.073073387145996, -0.8795639872550964, 0.30430397391319275, 1.0596299171447754, -1.2683923244476318, 1.245994210243225, 1.7227234840393066, -1.3516380786895752, 0.1869441270828247, -0.41785916686058044, 1.087266206741333, -0.047337356954813004, -1.0193865299224854, 1.0994023084640503, -0.5394179224967957, -0.5037531852722168, 0.20834527909755707, 0.13064995408058167, -0.4699561595916748, 2.0027852058410645, 0.8589077591896057, -0.5309668183326721, -0.2646041512489319, -1.2126028537750244, -0.6282755732536316, -0.030415114015340805, 1.0719408988952637, -0.1956489235162735, -0.38984596729278564, -0.2204776406288147, 0.9763778448104858, -0.8529828190803528, 1.0751609802246094, -0.42420727014541626, 0.1651744544506073, 0.48264947533607483, -0.06710823625326157, 0.5079697370529175, 1.8900227546691895, -0.691509485244751, 1.8339314460754395, -1.4819403886795044, -0.7274031639099121, 0.4721861481666565, 0.6261669397354126, -1.826695442199707, 0.8979287147521973, 0.5749494433403015, 1.3069045543670654, -1.7893558740615845, -0.14207150042057037, -0.7761998176574707, -0.6644382476806641, 2.712294340133667, 0.3535405993461609, -0.5252808928489685, 1.0148801803588867, -0.3468506336212158, 1.0109859704971313, 1.3427125215530396, 0.4754645526409149, -1.9957003593444824, 1.0261061191558838, -1.0228599309921265, 1.6614863872528076, 0.3104143738746643, -1.1170804500579834, 1.5777140855789185, -0.01517704501748085, 1.7217168807983398, 0.15462613105773926, 0.2867431342601776, -0.07223450392484665, 1.280221700668335, -0.1966591328382492, -0.29577696323394775, -0.7517362236976624, 0.5661882162094116, 0.6043401956558228, -0.009317469783127308, -0.6708378791809082, -2.0002760887145996, 0.026179654523730278, 0.966320812702179, 0.8855729699134827, -0.4442782998085022, -0.10245900601148605, -0.36523154377937317, 0.5373997092247009, -1.20217764377594, -1.2663779258728027, -0.5057381391525269, 0.7959527969360352, 0.07003208249807358, 1.2082902193069458, 0.36401253938674927, 1.4111645221710205, -0.3952288031578064, 1.1202036142349243, -0.2810516059398651, -1.6854645013809204, -0.533873975276947, -0.30895429849624634, -0.6466382145881653, -0.07815756648778915, 0.6864622235298157, 0.47576597332954407, 1.0314100980758667, 0.04632531479001045, -1.110945224761963, -0.6810529232025146, -1.485124945640564, -0.1088743731379509, -2.4302845001220703, -0.9249282479286194, 0.42798954248428345, 0.7537715435028076, 1.5733147859573364, -0.35580888390541077, -0.585422694683075, -0.8156148195266724, 0.11092951893806458, 0.36423587799072266, 1.1786720752716064, -1.3639788627624512, 0.9304026961326599, 1.5915857553482056, 0.6065329313278198, 0.5823027491569519, -1.1227478981018066, -1.666643738746643, 0.4550555944442749, 0.5779927968978882, 0.29503852128982544, 1.129899024963379, 0.06375010311603546, 0.4395408034324646, -1.8451433181762695, 1.0763291120529175, 0.44867947697639465, 1.1370370388031006, -1.2475614547729492, -0.688624382019043, 0.8494996428489685, 1.1701806783676147, 0.8218140006065369, 0.5047253966331482, 0.21454495191574097, 0.03694426640868187, 0.9952777028083801, -1.8725577592849731, 0.7414180636405945, 0.6937347650527954, -1.1759552955627441, -0.24828603863716125, -1.8944758176803589, -0.37614113092422485, -1.0689926147460938, -1.5753194093704224, 1.273140788078308, -0.6583883762359619, -0.8466948866844177, 0.4728873074054718, -0.7505086660385132, -0.2768947184085846, -1.0842307806015015, -1.2543423175811768, 0.9681221842765808, 0.42014816403388977, 0.10787276178598404, 0.9058806300163269, 0.5039677619934082, 1.4408588409423828, 0.013707969337701797, -0.6324089765548706, 0.9459042549133301, 1.395884394645691, 0.3992070257663727, -0.985650897026062, -2.1609725952148438, -1.3614344596862793, -0.9323062300682068, -0.4232613146305084, 0.3491286039352417, 1.2904038429260254, -1.6677321195602417, 0.5341277718544006, 0.8752820491790771, 0.7371194958686829, -0.5994114279747009, -0.5559695959091187, 1.1604517698287964, -3.852165460586548, 1.3093336820602417, 0.29037967324256897, -1.0597506761550903, -1.7991392612457275, -0.4783659279346466, -0.3370274603366852, 0.5182023644447327, 1.3727651834487915, -0.3392285406589508, -0.17671997845172882, 0.5287436842918396, 0.2528584897518158, 0.29694125056266785, 0.24357782304286957, -0.3000488877296448, 0.5040662884712219, -0.9376864433288574, 1.8292279243469238, -0.17040951550006866, 1.653876781463623, -0.9429283738136292, 0.6515690088272095, -1.1887876987457275, 0.1735764890909195, 0.5660556554794312, -0.10186631977558136, -1.0872204303741455, 0.35246357321739197, -1.31636643409729, 0.6815548539161682, -0.37226173281669617, 0.26908543705940247, -1.0928266048431396, -0.9216421246528625, 0.3235531747341156, -0.0995873212814331, -1.9049451351165771, 2.125230312347412, -2.2919535636901855, 0.7555210590362549, 0.13403533399105072, -0.8860841393470764, 0.45170342922210693, 1.2238543033599854, 0.7742140293121338, -0.4754756987094879, -1.5112160444259644, -0.010829566046595573, -2.456770420074463, 1.5067026615142822, -0.445852130651474, 0.43434804677963257, 0.8050114512443542, -1.8417140245437622, -0.9680891633033752, 0.22601155936717987, 0.42046889662742615, 0.12707507610321045, -1.728732943534851, 0.2547750174999237, 0.9706703424453735, -0.8063328266143799, -0.40588390827178955, -0.2173093855381012, -0.9521093368530273, 2.761732339859009, 0.6426706910133362, 0.8597468137741089, -0.9690319299697876, -1.2927618026733398, 0.4828975200653076, 0.5975548028945923, -0.43114736676216125, 0.5635680556297302, -0.458122581243515, 1.0139974355697632, -0.4966607093811035, -1.1107200384140015, 0.08650165796279907, -1.32437264919281, 0.4861702024936676, 0.6825398206710815, -0.5045374631881714, 2.6863551139831543, -0.3961365818977356, -0.40115928649902344, 0.19902515411376953, -1.0830756425857544, -0.6221067905426025, -0.3435022830963135, 0.7430158257484436, 2.200406312942505, -2.95154070854187, 1.4495793581008911, -0.39129894971847534, 0.4397670328617096, -0.9592656493186951, 0.8998770713806152, 0.859887957572937, 2.2495779991149902, 0.6174808740615845, -0.40784671902656555, -0.4957192838191986, -3.4715161323547363, -2.5907235145568848, 0.937604546546936, 1.4286670684814453, 0.0009908821666613221, -1.2931498289108276, -0.08814699202775955, 0.7992043495178223, 1.484811782836914, -0.7685871720314026, 0.28280892968177795, -1.0126272439956665, -1.464430570602417, -0.8121821284294128, -0.03718011453747749, 0.3593577444553375, -0.4054108262062073, 0.22251710295677185, 0.26203468441963196, 0.001970029668882489, -1.1788541078567505, 0.7318255305290222, 0.856705367565155, 0.4074380695819855, 0.296223521232605, -0.7322273254394531, -0.3025667369365692, 1.2357423305511475, -1.154700517654419, 0.9814739227294922, 2.5413103103637695, -1.2359219789505005, 0.38869795203208923, -1.622696042060852, 0.8523505926132202, -0.1095070019364357, -0.0927577093243599, -0.6556758284568787, -0.05396425724029541, 1.4777947664260864, -0.8745120167732239, 0.8328232765197754, -1.1805419921875, -0.17480000853538513, -0.5500271320343018, 2.552555561065674, 0.0697975605726242, -1.454836368560791, 0.05165252834558487, -0.7944586277008057, 1.120506763458252, -1.9049474000930786, -0.1925811618566513, 0.9170861840248108, -1.0458675622940063, -1.0233147144317627, 1.3199056386947632, 0.19758287072181702, -0.6528744101524353, 0.24552659690380096, -0.535018265247345, 0.4512055814266205, -1.5123308897018433, -0.5321947336196899, -1.3416694402694702, -0.7012051343917847, 2.5942392349243164, -1.7452130317687988, -1.4883275032043457, -0.7899966239929199, -1.3510252237319946, -1.307806134223938, 0.6534383296966553, 1.1191167831420898, 0.5105189085006714, 1.2082023620605469, -3.593231201171875, 0.043784432113170624, -0.7302325367927551, 0.42270562052726746, 0.03294806554913521, -0.3631855845451355, 0.16129247844219208, 0.015703294426202774, 1.5872166156768799, -1.3909494876861572, 2.2693138122558594, -0.2708352506160736, -0.2146226167678833, 0.14034157991409302, -0.38632750511169434, 0.3575234115123749, -0.0814099833369255, 0.41292834281921387, -1.717434287071228, -0.9078443050384521, -0.8624849319458008, -1.0391765832901, 0.5023599863052368, 0.766029953956604, -1.2133806943893433, 0.6713598966598511, -1.2586373090744019, -1.9028033018112183, -1.007853388786316, 1.243224859237671, 0.2805533707141876, 0.5916670560836792, 0.10244731605052948, 0.4051184356212616, 0.07962394505739212, -1.3571388721466064, 0.10906961560249329, -1.1434558629989624, 0.03537662327289581, 0.5436141490936279, -1.297612190246582, 1.505203366279602, -0.21498987078666687, 1.5337871313095093, -0.47894686460494995, 0.24056243896484375, -0.2057839184999466, 0.6749655604362488, 1.0473824739456177, 0.7499487400054932, -0.5096343755722046, 1.608832597732544], [1.5401197671890259, -0.5491004586219788, 0.7778387069702148, 0.7469237446784973, -0.6298142075538635, 0.8087716102600098, -1.3297430276870728, 1.1867575645446777, 0.45768484473228455, 1.1262437105178833, 0.12763474881649017, -0.017926551401615143, -0.3530576825141907, 1.4212040901184082, 0.9942249655723572, 0.6118272542953491, -0.21171268820762634, 2.2210452556610107, 0.16899892687797546, -1.6011916399002075, 1.4953556060791016, 0.7553986310958862, -0.41110363602638245, -2.0462911128997803, 0.3894886076450348, 0.16611385345458984, -0.13779784739017487, 1.1228810548782349, -1.5705713033676147, -0.6220701932907104, -1.23415207862854, -1.291395902633667, -0.6318552494049072, -2.0285675525665283, -0.9112683534622192, 0.7239915728569031, -0.0852171778678894, 0.4336502254009247, -0.15598247945308685, -0.07932554185390472, -0.6699812412261963, -0.40100598335266113, 0.16931389272212982, 1.0857677459716797, 0.02301022782921791, 0.07995103299617767, 0.14352214336395264, -0.5273150205612183, 0.4071812033653259, 1.2149837017059326, -0.7750453352928162, -0.7644844055175781, 0.9737403988838196, -0.9493513703346252, -0.9060689806938171, 1.0783030986785889, -0.8110003471374512, -0.705752432346344, -1.3064159154891968, -1.1783757209777832, 1.9244091510772705, 2.547081232070923, -0.5272589325904846, 0.41792312264442444, 1.4297020435333252, 0.12222035974264145, 0.4840848445892334, -0.747627854347229, -0.7296427488327026, -0.14617189764976501, -1.222357153892517, -0.47637543082237244, -1.987004280090332, 0.9591853618621826, 1.3860478401184082, -0.7952947020530701, -0.5587637424468994, -0.5162286758422852, 1.4274758100509644, 2.107025146484375, -0.3872077465057373, -1.1040316820144653, 0.10271059721708298, 0.8737886548042297, 1.4845690727233887, 0.4299362003803253, -0.7883785367012024, 0.5007784962654114, 0.5770206451416016, 1.3936458826065063, -0.6977832317352295, 0.0265607051551342, -0.5841132998466492, 0.5223527550697327, -0.9362531304359436, 0.4478360414505005, 0.7252258062362671, -1.4669462442398071, -0.8279346823692322, 0.2522854208946228, -0.898431658744812, 0.42326590418815613, 0.21465036273002625, 0.5063678026199341, 0.7057658433914185, -1.0203328132629395, -0.4804917871952057, -2.2685084342956543, 0.030024411156773567, -1.6848641633987427, 1.5774226188659668, -0.1593824177980423, 2.0216877460479736, -0.3776370882987976, -0.30200478434562683, -0.3120516836643219, 0.03043493814766407, 0.6135185956954956, 0.19806717336177826, -2.2764439582824707, -0.32032862305641174, -0.043870244175195694, -0.03735256940126419, 0.7728177905082703, -0.9668079614639282, 0.0012415562523528934, 0.4089376628398895, 0.4710054397583008, -0.031530581414699554, 0.8570906519889832, 0.05495024099946022, 0.29885333776474, -0.6115185618400574, -0.09780479967594147, -1.014026165008545, -0.41996362805366516, -1.2665055990219116, -1.2986027002334595, 0.9324052929878235, 0.7175065279006958, 0.2144274264574051, 0.6426364779472351, 0.13218723237514496, -0.29449015855789185, 1.4104893207550049, -1.1794698238372803, -0.14191962778568268, -0.1810327023267746, -0.043297991156578064, -0.32860589027404785, 0.40444833040237427, 1.7863118648529053, -0.9585207104682922, 1.0313242673873901, -0.1157739907503128, 0.6009297966957092, -0.1627255082130432, 1.2224969863891602, -0.9568083882331848, 2.3674991130828857, -0.6063680052757263, -0.2789236605167389, -0.5561293363571167, 0.5756704211235046, -3.2069828510284424, 0.30262109637260437, -0.5918357372283936, 1.3952909708023071, -0.6696699261665344, 1.4124274253845215, 1.1638720035552979, 1.8625670671463013, -3.085963010787964, 0.6771887540817261, -0.42593252658843994, -0.34061458706855774, -0.12647853791713715, 0.7732170224189758, -0.20819033682346344, 1.49164879322052, 2.5338592529296875, 1.2048717737197876, 0.20973928272724152, 0.8432213068008423, 1.0989936590194702, -0.0963946282863617, 0.38830286264419556, 0.6554950475692749, 0.02727774903178215, 0.8499489426612854, -1.6584618091583252, -1.4275116920471191, -0.8758272528648376, -3.861528158187866, 0.6214864253997803, -0.5151300430297852, 0.6534560322761536, -0.6389850378036499, 0.5287356972694397, -0.8509553074836731, -1.0773664712905884, 0.6545868515968323, -0.5417629480361938, 0.8711471557617188, -0.3155210018157959, 0.37225791811943054, 0.4786057472229004, -0.45227929949760437, -0.6878065466880798, -1.0537070035934448, -0.28734496235847473, -1.0825185775756836, -0.13261768221855164, -0.799029529094696, 0.3476433753967285, -0.6466544270515442, -0.9858987927436829, 0.5468904972076416, 1.3043254613876343, 0.0062436675652861595, 0.1637030392885208, 0.8277908563613892, 1.1951402425765991, -0.6131725311279297, -2.0227510929107666, -0.06204375624656677, 1.3807554244995117, -1.252343773841858, 2.2228455543518066, 2.273552179336548, -1.485344409942627, 1.1522469520568848, -2.2119956016540527, 0.6275163888931274, 0.6195469498634338, 0.6837811470031738, 1.2472679615020752, -0.17574253678321838, 1.3128334283828735, -0.9412449598312378, -0.7163158655166626, -0.49799710512161255, -0.0156287532299757, 1.4153916835784912, -0.03028310090303421, -0.6953220963478088, -0.38512009382247925, -0.679520845413208, -0.5235272645950317, 0.33952584862709045, -0.12060526013374329, 0.7741318345069885, 1.1227566003799438, -0.07544009387493134, 0.7368286848068237, -0.04652564972639084, -0.9751104712486267, -1.1630452871322632, 1.1543407440185547, 0.1180194541811943, 0.5627546310424805, 0.5370272994041443, -0.19775395095348358, 1.4393131732940674, 1.376475214958191, 0.2972905933856964, 0.12489701807498932, -0.28188079595565796, -0.7735673785209656, -1.0343670845031738, 0.036076582968235016, -1.747153401374817, 0.03698471188545227, -0.14466746151447296, -0.920242965221405, 0.6402252316474915, 0.8059341907501221, 0.23519587516784668, 0.8183708190917969, -0.07189983129501343, -0.9231435656547546, 0.8890631198883057, -1.1426451206207275, 1.5816192626953125, 0.8203930854797363, 1.4381998777389526, 1.2364603281021118, -0.3074013590812683, 1.1877943277359009, 0.9809601902961731, -1.5408499240875244, -0.639244794845581, 1.3752998113632202, -0.6477894186973572, -1.7849172353744507, 0.09426312893629074, -0.3014419972896576, 1.870874285697937, 0.3021349608898163, -1.1383541822433472, 0.19344209134578705, 1.215081810951233, 0.1032227873802185, 1.3569469451904297, -1.4538414478302002, 0.4371402859687805, 0.33793190121650696, 0.5737248063087463, -0.18572615087032318, -0.09815976768732071, -0.3793196976184845, -1.5732396841049194, -0.05663437396287918, 0.1658540517091751, 1.0645209550857544, 0.47955116629600525, -0.03649505227804184, 0.5192960500717163, -0.12194181233644485, 0.489276260137558, 0.709184467792511, 1.1387637853622437, 0.08939504623413086, -0.1094549149274826, -0.9098944664001465, 1.7517004013061523, 1.256609559059143, -0.5294193625450134, 0.042129501700401306, 0.2919818162918091, -0.4195388853549957, -0.1811223030090332, -0.07920384407043457, -0.9996359348297119, 0.7971836924552917, -2.075632095336914, -0.2194206416606903, 0.45403704047203064, 0.29554852843284607, -0.4131593108177185, -0.2655014991760254, -0.08448304980993271, -0.14776545763015747, 1.5615253448486328, 0.15088872611522675, 1.8607981204986572, 0.2940182089805603, 1.3974615335464478, -0.17616978287696838, 0.5638839602470398, 0.14238667488098145, -1.023450493812561, -0.9095591306686401, 0.5865501165390015, -0.9108880162239075, -1.800642728805542, 1.6283010244369507, -0.6919599175453186, 2.3160643577575684, 0.18728840351104736, 0.08085546642541885, 1.7866138219833374, 1.1308014392852783, 0.5426053404808044, 0.2861367166042328, 0.43517017364501953, 2.1466963291168213, 0.5041483640670776, 0.5160337090492249, 1.4131507873535156, -0.4259520471096039, -0.8461704254150391, 0.4418134093284607, -0.4196498990058899, 1.1584481000900269, 0.015664823353290558, -1.7099733352661133, 0.25845685601234436, -0.8187965154647827, 0.8600371479988098, 0.0717795118689537, 1.5834866762161255, -0.3716849088668823, 0.47662878036499023, -0.871040940284729, 0.4927894175052643, 1.6262503862380981, -0.007911463268101215, -0.6501232385635376, -0.0936901643872261, 1.1484854221343994, -1.3685110807418823, 0.26807907223701477, 0.46174299716949463, 0.0830775797367096, 1.4589520692825317, -0.11645497381687164, 0.33213019371032715, 0.3003397285938263, -0.5123298764228821, 0.24896694719791412, -0.23852886259555817, -0.22484846413135529, 0.05754859372973442, -0.9367002248764038, 0.6619271039962769, 0.8107920289039612, -0.23397304117679596, 0.886955738067627, -2.3252267837524414, -0.3569696247577667, 1.2198233604431152, 0.6210957169532776, 0.24114525318145752, -0.02735585905611515, 0.31329289078712463, 0.972241997718811, 2.3968796730041504, -1.3396834135055542, -0.7489110231399536, 0.3713013231754303, -0.5900385975837708, 1.0924243927001953, 1.1671051979064941, 0.9894277453422546, 0.4379468262195587, 1.0725069046020508, 1.2735958099365234, 0.1724964678287506, 0.14328932762145996, -0.6244006156921387, -1.5364868640899658, -0.2557394504547119, 1.0492868423461914, 2.4906673431396484, -0.728358268737793, -0.9757213592529297, 0.7367355823516846, 1.1029763221740723, 0.7581115365028381, 1.4191936254501343, -0.23130813241004944, -0.10406361520290375, 1.240319848060608, 0.7151231169700623, 0.9024981260299683, 2.4264743328094482, 0.5760591626167297, 0.9827942252159119, -0.3535975217819214, -0.12807993590831757, -0.186215341091156, -0.42308661341667175, -0.5955061912536621, 0.05380682647228241, -1.452799677848816, -0.3199688196182251, -0.35189542174339294, -0.3627212345600128, -0.2719605565071106, -1.5358564853668213, -0.07827621698379517, 1.5593914985656738, 0.48936721682548523, 0.290479451417923, 0.3536887466907501, 0.5473164916038513, -0.3865486681461334, -0.6328052282333374, 0.8377465605735779, -0.06961629539728165, -1.291856050491333, -0.22728466987609863, 1.234125018119812, -0.23532308638095856, 0.975365161895752, 0.9564276933670044, 0.34627771377563477, -2.2146313190460205, -0.03353509679436684, -0.40943875908851624, 0.7391632199287415, -0.7188723087310791, -0.3766115605831146, -0.39523112773895264, -0.2479410320520401, 0.822365939617157, -0.3092300295829773, -0.1340097337961197, -0.8737068772315979, 0.08002358675003052, -0.11881175637245178, -0.6192734241485596, 1.9134643077850342, 0.32769864797592163, -0.2850843369960785, -0.5637667179107666, -0.927241325378418, -0.287007600069046, 1.5293529033660889, 0.0037260903045535088, -1.5013322830200195, 0.6037412881851196, -0.22050189971923828, 1.261490821838379, 1.5799901485443115, 1.5769654512405396, -0.7398552298545837, -0.5939614176750183, -0.7272220849990845, 0.9186690449714661, 1.1261898279190063, -0.6469132900238037, -0.13113559782505035, 0.71834397315979, -0.5323953628540039, 1.1783013343811035, -0.7157368063926697, 1.4613398313522339, -1.7235227823257446, -0.41076454520225525, -0.9286868572235107, -0.08376222103834152, 1.2266608476638794, 0.389724463224411, 0.06311219930648804, 0.1771399974822998, 0.6665477156639099, 0.502915620803833, 0.408482164144516, -0.17121174931526184, 0.6437785029411316, 0.3304116129875183, -0.4500385820865631, 0.011109787039458752, 0.4708574116230011, -1.052047610282898, -0.45511651039123535, 0.11259864270687103, -0.9760291576385498, 0.7852494716644287, -1.2712072134017944, -0.22346746921539307, 0.8611302971839905, 0.38284316658973694, 1.4768925905227661, -0.17922888696193695, -0.70545893907547, -0.17678192257881165, 0.36692407727241516, 2.311479330062866, 0.07552164793014526, -0.23154762387275696, 0.4181171953678131, -0.8387276530265808, -0.9894188642501831, 0.41336238384246826, -1.412053108215332, 2.090955972671509, -1.566011905670166, 0.5779523253440857, 1.0677614212036133, -0.9286702275276184, 0.3463270366191864, -0.09445691108703613, -0.5405825972557068, 0.335199236869812, -0.49747124314308167, -1.2568124532699585, 0.051758140325546265, 0.32888105511665344, -0.23408645391464233, -0.37015730142593384, 0.39850470423698425, 0.07856196165084839, -0.6666789054870605, -0.08690184354782104, 0.32905739545822144, -0.8525896668434143, 1.3921210765838623, -2.073239803314209, -1.3870760202407837, 0.5775946974754333, 0.01960648223757744, -1.693972110748291, -1.2425490617752075, -0.6217936277389526, 0.8794918656349182, -0.1127345934510231, -1.5275307893753052, -0.9724931120872498, -1.3822314739227295, -1.4136182069778442, 1.0018227100372314, -0.10619427263736725, -0.9769010543823242, -0.985069990158081, -1.075402855873108, 0.3761135935783386, -0.0814087763428688, 1.146906852722168, -0.3153947591781616, 0.20130670070648193, 0.32868021726608276, 0.5697015523910522, -0.4716803729534149, 0.33434879779815674, 0.06871239840984344, -1.0319424867630005, -1.082664132118225, -0.529662013053894, -1.471641182899475, -0.2035883665084839, -1.2170473337173462, 2.5340678691864014, 0.20585498213768005, 0.4012100398540497, -1.2831186056137085, -0.5520204305648804, 0.046599481254816055, 0.4908493161201477, 1.2990407943725586, 1.7581371068954468, 0.8031694293022156, 0.38816937804222107, 1.12028169631958, 0.9253832697868347, 0.2676035463809967, 0.24376265704631805, 0.1474730223417282, 1.6243202686309814, -0.2563697397708893, 0.023662561550736427, -2.46006178855896, -0.16513486206531525, -0.8699630498886108, 0.23192310333251953, 0.8246321082115173, -2.1015758514404297, -0.42451155185699463, 0.7245345115661621, -1.6757768392562866, -0.8847615122795105, -0.9334514737129211, 0.9422733783721924, -0.6798949241638184, -0.4004492461681366, -0.56546550989151, -1.2128242254257202, 1.2169032096862793, 1.8478540182113647, -0.7246072292327881, 2.6204960346221924, -0.4142071604728699, -0.8010064363479614, -0.9921383857727051, 0.23949263989925385, -0.429319828748703, 0.04442442208528519, 0.8016964793205261, -0.09276513010263443, 0.567646324634552, -0.16016942262649536, 1.1765556335449219, -0.09734000265598297, 1.3681102991104126, -0.16900566220283508, 1.1298831701278687, 0.45471957325935364, 0.5466572046279907, -0.18127040565013885, 0.39725232124328613, 0.48690181970596313, 2.1273787021636963, -1.4585137367248535, -0.6210675239562988, 0.8455579280853271, -1.0455950498580933, -0.2825852632522583, 0.5204891562461853, 0.5119758248329163, -2.4153976440429688, 1.3199357986450195, 2.0501821041107178, -1.336485743522644, 1.0445612668991089, 0.7308616638183594, -1.0568093061447144, 0.3113090693950653, 1.7831214666366577, -0.08361973613500595, -0.38414108753204346, 0.3681349456310272, 0.8133945465087891, 0.14369173347949982, -0.731062114238739, 1.3286839723587036, 0.2941541075706482, -0.8075625896453857, -0.9184131622314453, 0.4670698642730713, 2.1980109214782715, 1.941051721572876, -0.6123737096786499, 0.5900260806083679, 2.684424877166748, -1.5450189113616943, -0.2800537049770355, 0.5309045910835266, -0.287803590297699, 0.5087596774101257, 0.8249565958976746, -0.1612735241651535, 0.5139696002006531, -0.07452704757452011, 0.015393373556435108, -0.33170565962791443, -2.253751754760742, -0.5030736327171326, 1.8258527517318726, 0.7890871167182922, -0.08221794664859772, -0.23714008927345276, 0.5818670988082886, -0.3885795772075653, -0.8864502310752869, -0.4066607654094696, -0.14601856470108032, 0.2894427180290222, -0.24933698773384094, -0.1695265918970108, -0.6202830076217651, -1.021627426147461, 0.3849540650844574, -2.369291305541992, -0.7806940078735352, -0.2908630073070526, 0.8642311692237854, -0.2260676771402359, -0.8041931986808777, -0.37940365076065063, 1.2356343269348145, -0.8597760796546936, 0.9251825213432312, 0.19381621479988098, 1.6601425409317017, -1.057517647743225, 0.1510278880596161, 0.35400086641311646, 1.7294962406158447, 0.5111709833145142, -1.2578387260437012, 0.7429134845733643, -1.3661662340164185, 1.9299137592315674, 2.1436996459960938, 0.3427228331565857, 0.3802759051322937, -0.4798987805843353, 0.1806703805923462, 0.21409453451633453, 0.6207169890403748, -0.9888989925384521, -0.07231676578521729, -0.8319811224937439, -0.8048908710479736, -0.1984459012746811, 0.03340764716267586, -0.4659271538257599, -1.635723352432251, -0.04962639883160591, 1.8731353282928467, -0.3292291462421417, 1.0790352821350098, -0.0573759526014328, 0.3559338450431824, -0.08669356256723404, 0.16839846968650818, 0.6721559166908264, -0.5985718965530396, -0.5317187905311584, -2.0430350303649902, -1.0053895711898804, 0.696922779083252, -0.7350214123725891, 0.48989439010620117, 0.8284916281700134, 0.8882510662078857, -0.6975094079971313, 1.0988045930862427, -0.06658901274204254, -0.26199132204055786, -1.1985422372817993, 1.191752552986145, -0.03285451605916023, 0.3957318365573883, -2.50736665725708, 1.5132358074188232, -1.5934025049209595, -1.6537333726882935, -1.3288229703903198, -0.2157277911901474, 0.14644481241703033, -1.1380233764648438, -0.18326126039028168, -0.36514467000961304, -0.09070077538490295, -0.5703496336936951, -0.4056451916694641, 1.2168443202972412, 0.8712696433067322, -1.778927206993103, 1.5358482599258423, -0.28607064485549927, 0.872226893901825, -0.4115388095378876, 0.6795371770858765, 0.47221505641937256, -0.6379454135894775, 0.2363225370645523, 0.018584301695227623, -0.5428816080093384, -1.0850468873977661, 0.9579108357429504, 0.4703206717967987, -1.0008087158203125, -0.013752459548413754, -1.1430068016052246, -0.7275763154029846, 1.540229082107544, 1.1410995721817017, 0.11099319159984589, 1.6849050521850586, 0.8828701972961426, -1.5481233596801758, 0.33093154430389404, 0.22196970880031586, 0.6427333950996399, 0.3026657998561859, -1.244848370552063, 0.4924420118331909, 0.977157711982727, 0.025496458634734154, -0.13312391936779022, 1.9180302619934082, -2.6898515224456787, -0.5238210558891296, 0.6439749002456665, 1.6743932962417603, -1.7847659587860107, 0.2658836841583252, 1.2802854776382446, -0.48952245712280273, -1.8746259212493896, 1.1393073797225952, -0.3857823610305786, 1.2079803943634033, 1.0296354293823242, 1.0314991474151611, 0.2996855676174164, 1.198563575744629, 1.6802889108657837, -1.4301447868347168, 1.2327758073806763, -0.6056979894638062, -0.785290539264679, -0.3923731744289398, -2.127715587615967, -0.246408149600029, 0.013118655420839787, -0.3024488091468811, -1.1272404193878174, 0.4268093407154083, -0.02915291301906109, -2.7470414638519287, 0.5655549764633179, -0.013078478164970875, -0.2222258597612381, 0.6974546313285828, 1.1710413694381714, -2.3086984157562256, -0.07468753308057785, 1.6273553371429443, -0.901035487651825, -0.13815870881080627, -0.02482195384800434, -0.6808879971504211, 0.9238810539245605, 0.26202264428138733, 0.6137514114379883, 0.6401759386062622, -0.5999636054039001, -0.010856172069907188, 0.2563398778438568, 0.18207165598869324, 1.8272300958633423, -0.12568214535713196, 2.6713736057281494, 1.027462363243103, 1.6200716495513916, -0.4281514585018158, -0.22345873713493347, -1.1083849668502808, 0.8192078471183777, -0.26875099539756775, 0.4146176278591156, 0.4246055781841278, 1.451230764389038, -0.1779828667640686, 1.0104122161865234, -0.11928772926330566, -0.45852911472320557, 1.6029685735702515, -0.032214704900979996, 0.21687771379947662, 1.1918309926986694, -0.7843013405799866, -1.0141806602478027, 0.21641318500041962, 0.5373764038085938, 0.5139298439025879, 2.3947012424468994, 0.8485471606254578, 1.0249526500701904, -1.2757962942123413, -0.6529989838600159, -1.110474944114685, 1.0904349088668823, -1.125797152519226, -0.5717002153396606, -1.3523995876312256, 1.4895271062850952, -0.5428513884544373, 0.8224903345108032, 1.5795625448226929, 0.48940950632095337, -0.7568188309669495, -1.0209496021270752, 0.029585029929876328, -0.413979709148407, 0.24831941723823547, 0.5827746391296387, -0.294399231672287, -0.6007898449897766, -0.7401251792907715, -1.755629062652588, 0.5031570792198181, -1.2903459072113037, 0.11658673733472824, 0.032174695283174515, -0.32422924041748047, 0.8692352175712585, 0.2591751217842102, 1.8037229776382446, 0.4057011604309082, 1.0489753484725952, 1.0272091627120972, 1.3815054893493652, -0.429364949464798, 2.6313610076904297, -2.266775131225586, 0.545799732208252, 0.45161172747612, 0.5841052532196045, 2.1819908618927, -1.708406925201416, 0.7613571286201477, -0.8410559892654419, 1.5681875944137573, 0.8740506172180176, 1.1209259033203125, -0.534134566783905, -1.1794919967651367, 0.5287038087844849, -0.3707989454269409, -0.6911197304725647, 1.4464582204818726, -1.1199291944503784, -0.7573873996734619, -0.5065471529960632, 0.7462963461875916, -1.0918614864349365, 0.04770991578698158, 0.20966538786888123, 0.7215152382850647, -0.6848055124282837, -1.9325976371765137, -0.5897459387779236, -2.412213087081909, -0.341886967420578, -1.8336589336395264, -1.9492294788360596, -0.9016523957252502, 1.62649405002594, -1.879201054573059, -0.21493951976299286, -0.4454900324344635, 1.4752871990203857, -0.4335225224494934, -0.2482975721359253]], "dtype": "float32"}, 1], "e": {"__nd__": [-35.09896469116211, -82.79090118408203, 17.724645614624023, -53.93408966064453, 45.33815002441406, -17.376920700073242, 8.303396224975586, -62.56935119628906, -84.38884735107422, -36.44257736206055, 66.78057861328125, 11.486628532409668, 20.874990463256836, -2.8635008335113525, -24.320085525512695, -56.60329055786133, 33.124271392822266, 25.47136116027832, 30.1877498626709, 9.066948890686035, 9.732592582702637, -39.39942169189453, -7.722761631011963, -26.229032516479492, 37.19306564331055, -0.9033054709434509, 28.21783447265625, -8.84988021850586, 11.434310913085938, -8.772080421447754, -32.88547134399414, 1.158867359161377, 20.023378372192383, -28.759197235107422, 12.830245971679688, -66.65440368652344, 52.06760025024414], "dtype": "float32"}}, {"a": [{"__nd__": [[0.2065650373697281, 1.4194836616516113, 0.15209156274795532, -1.6393558979034424, -0.17635098099708557, 0.9750550389289856, 1.2269902229309082, -0.7543753981590271, -0.1959836184978485, -0.18712998926639557, 0.03907274082303047, 0.9746937155723572, 0.17471511662006378, -0.346910685300827, -0.5975297689437866, 0.857040524482727, -0.6581949591636658, 1.1586425304412842, 0.24953308701515198, -1.7784240245819092, 0.32281166315078735, -1.974778652191162, 0.2121279388666153, -0.522205114364624, 0.563452422618866, -0.5358158349990845, 1.318186640739441, -0.3699449598789215, 1.5988128185272217, 0.2077072113752365, 0.19548740983009338, 0.15945695340633392, -0.5694707632064819, 1.0248478651046753, -2.522148370742798, -0.4532361626625061, -0.5236812829971313, 1.5229361057281494, -1.4245482683181763, 0.097941093146801, 1.1143397092819214, -0.8157839179039001, 1.5110790729522705, -0.11773961782455444, -0.6739119291305542, -0.8132210969924927, -1.1860966682434082, -0.15363147854804993, -0.3408355116844177, -0.05433463305234909, -1.463974118232727, 0.7922946810722351, 0.11091314256191254, 0.3421538174152374, -0.6279222965240479, 0.01541126612573862, -0.05580738186836243, 1.046299934387207, 0.9935141205787659, -0.10741913318634033, -0.6551379561424255, 1.8107448816299438, -0.5755385756492615, -0.9738866090774536, -0.4941185712814331, 1.7451536655426025, -0.5932671427726746, 0.948929488658905, -1.2454243898391724, 1.1736489534378052, 1.7043527364730835, -0.6079495549201965, -0.8313103914260864, -0.5643629431724548, 0.21282215416431427, -0.32670724391937256, 0.9178809523582458, 0.7009218335151672, 2.158442497253418, 0.9253513216972351, 1.2789552211761475, 0.9332144856452942, -1.8252723217010498, -0.44741135835647583, 0.7448534965515137, 1.91130793094635, 0.2600846290588379, -0.8286184072494507, 0.9799556732177734, -0.7287525534629822, 0.36664924025535583, -1.1295546293258667, 0.6522035598754883, -1.0761879682540894, 1.3712772130966187, -0.6643219590187073, 0.5387807488441467, -0.6528935432434082, 0.30651286244392395, -0.11676305532455444, 0.615185022354126, 1.081640362739563, -0.4277217388153076, 1.0038564205169678, -2.328233242034912, -0.12203668057918549, 0.40886110067367554, 0.7146164178848267, -1.4731543064117432, 0.8560152053833008, 1.123231291770935, -1.3815147876739502, -0.3021200895309448, 0.6578772664070129, -0.4827588200569153, 1.0642459392547607, 1.0078343152999878, -0.48650550842285156, -0.6462295651435852, 0.8186724781990051, -1.5178494453430176, -0.3575875461101532, 0.6502118110656738, -0.7815431356430054, -0.07314636558294296, 0.6321502327919006, -1.699236512184143, -0.6940560340881348, -0.5395640730857849, 0.31468719244003296, 1.435336709022522, -1.2701407670974731, -0.675449788570404, -0.3010879158973694, 0.8106183409690857, 0.4583727717399597, 1.0269927978515625, -1.0853252410888672, 0.24746796488761902, -0.3092373013496399, -0.036779992282390594, -1.3103994131088257, 1.2270950078964233, -0.9369661211967468, -0.16083768010139465, 1.148123860359192, 0.595869243144989, 0.14446164667606354, 1.1146793365478516, 0.12244226038455963, -0.2059326320886612, -0.016820799559354782, 0.19859741628170013, -3.1537973880767822, -0.12243146449327469, -0.525398313999176, 0.21830910444259644, -1.101082682609558, 1.2499816417694092, -0.24557586014270782, 1.0036838054656982, 0.6317331790924072, 0.8358795046806335, 0.09938672930002213, 0.42441311478614807, 0.3267926275730133, -1.7990740537643433, -0.523358166217804, -0.3884701430797577, -1.4391491413116455, -1.3091166019439697, 0.5991506576538086, -0.8455089330673218, 0.4984554052352905, -0.548332929611206, 0.06761269271373749, 0.28905948996543884, 0.17626258730888367, 0.6833481788635254, -0.129084512591362, 0.0784216970205307, -0.5042449235916138, 1.9813764095306396, -0.8769988417625427, 1.2035584449768066, 0.635132372379303, 1.030734658241272, -0.7989813685417175, 0.4313599169254303, -0.7314174175262451, -1.0041497945785522, 1.6557368040084839, 0.37529733777046204, -0.6712630391120911, 0.16515658795833588, -0.4775511920452118, 0.8079532980918884, -0.9598753452301025, -1.0362251996994019, 0.24284908175468445, -1.5312554836273193, 0.5579637289047241, -0.8621180057525635, 0.26592621207237244, -0.5958254933357239, 0.7654720544815063, 0.12616291642189026, 0.41756394505500793, -2.0300631523132324, 0.08272265642881393, 0.11030115187168121, 0.6832738518714905, 1.1367918252944946, 0.47129884362220764, -0.4429212212562561, 0.6416391134262085, 0.6455983519554138, 0.12326265871524811, 1.6881775856018066, 1.592786431312561, 0.8814534544944763, -0.02078264020383358, -0.8382565975189209, 0.0810975432395935, 0.702067494392395, -2.267228841781616, 0.9159107804298401, 0.5544248819351196, -0.04865832254290581, -0.14346933364868164, -1.0526635646820068, 0.22152747213840485, -0.8935034275054932, 0.7424785494804382, 0.008954011835157871, -0.05147486552596092, 1.2162021398544312, 0.4409787952899933, 1.3694064617156982, -0.09007863700389862, -0.8596698641777039, 0.10622230917215347, -0.37181058526039124, -0.0067849718034267426, 0.4637286961078644, 0.3742583692073822, 0.35876843333244324, -1.0117900371551514, -0.12225867062807083, -0.08242329955101013, 0.1649305820465088, 0.22167372703552246, 1.7772246599197388, 0.19024117290973663, -0.5213483572006226, -0.5007416009902954, -0.9087387919425964, 0.18545086681842804, 0.07837074995040894, 2.575859785079956, -0.2388215959072113, -0.27764075994491577, -0.3535749316215515, -0.09585890918970108, 1.4663360118865967, -1.2548325061798096, 0.4808584749698639, 0.9099956750869751, -0.13804693520069122, 0.30973708629608154, -0.8437023758888245, 0.025666818022727966, 0.3268410563468933, -0.49772170186042786, -0.8732519149780273, 0.5803821086883545, -0.023419557139277458, 1.0713202953338623, 0.3002952039241791, -0.25077322125434875, 0.6513085961341858, 1.1936020851135254, 0.5336251258850098, -0.8338047862052917, -0.2888035476207733, 1.0317529439926147, 0.4897615313529968, -0.08238628506660461, -0.30182936787605286, -1.398168921470642, -0.04159785434603691, -0.8062137365341187, -0.5881291627883911, -0.1431388109922409, -1.459970235824585, -0.2443370372056961, 1.4043248891830444, 0.18161466717720032, 1.5659633874893188, -0.020557019859552383, -0.07763616740703583, 0.6709205508232117, -0.5517421960830688, -0.3340725898742676, -0.706313967704773, 1.0473276376724243, -0.7874857783317566, 0.21872058510780334, -0.2560131847858429, -0.7327775359153748, 1.9032816886901855, -0.6738521456718445, -1.245646357536316, 0.24192066490650177, 0.7013838291168213, 0.19666871428489685, -0.7218449115753174, 0.40868470072746277, 2.0638840198516846, -0.11951308697462082, 0.00864381156861782, -0.36727645993232727, -0.19187991321086884, -0.06467342376708984, 1.4514813423156738, 1.3750653266906738, 0.09106782078742981, -1.2653459310531616, -0.3919163942337036, 0.2767956852912903, -0.14456544816493988, 1.8270870447158813, -0.8589468598365784, 1.0971696376800537, -1.484216570854187, -0.7610009908676147, 0.7177760004997253, 0.6873390674591064, -0.21670465171337128, 0.022728731855750084, 2.2587945461273193, 1.391963005065918, 2.2993311882019043, 0.2690458595752716, 1.59824538230896, 0.10012780129909515, 0.2960290312767029, 1.358752965927124, 0.16819502413272858, 1.113067388534546, -0.05485493317246437, 1.4640400409698486, -0.6390298008918762, 0.36274453997612, -0.25804364681243896, -0.6212397217750549, 0.24566885828971863, 0.5895450711250305, -0.17846594750881195, 0.49103862047195435, 1.1856166124343872, 0.24166448414325714, -0.4082431197166443, -0.8687567114830017, -0.14272440969944, -2.123016357421875, -0.4159659445285797, -3.1293909549713135, -0.7236876487731934, -1.1947126388549805, 0.789694607257843, 0.10050775110721588, 0.29632532596588135, -0.18706932663917542, 0.6922820210456848, 0.5008264183998108, 0.8714815378189087, 1.14119291305542, 0.7178844809532166, -0.8933191299438477, 0.8368484973907471, -1.3398247957229614, -1.8061504364013672, -0.18836891651153564, 1.4185948371887207, 1.2832452058792114, 0.3677048683166504, -1.6883639097213745, -0.9516795873641968, -1.1694034337997437, 0.22139430046081543, 1.1544846296310425, 0.5028183460235596, -1.3080333471298218, -1.2819175720214844, -1.1831895112991333, 1.2033358812332153, 0.8006065487861633, 2.005485773086548, -1.0998603105545044, -1.248868465423584, 1.3479106426239014, -0.9345016479492188, -0.06759605556726456, 0.3217306435108185, 0.8012588620185852, -1.2593815326690674, 0.7788556218147278, -0.5463085174560547, 1.778515338897705, 0.6009320020675659, 1.4363138675689697, 2.5825252532958984, -1.127663254737854, 0.47929030656814575, 1.3639869689941406, 1.472808837890625, -0.6110773086547852, -1.3345640897750854, 2.146145820617676, -0.5197583436965942, -0.3927359879016876, 1.4680440425872803, -0.5486389398574829, 1.754269003868103, -0.3235055208206177, 1.3740041255950928, -0.5537219643592834, -1.51162588596344, -0.5362884998321533, 0.41070955991744995, -1.4173372983932495, 0.24721960723400116, 0.11723609268665314, -0.7864993810653687, 0.6788805723190308, -0.4492645859718323, -1.054115891456604, 0.14649923145771027, 0.8472549319267273, 0.14238408207893372, -0.5082601308822632, 0.67425537109375, -1.2222965955734253, -0.868867039680481, 0.5524448156356812, 0.6864125728607178, 0.6022374629974365, 0.717625617980957, -1.0443170070648193, -1.5154836177825928, 1.9397687911987305, -0.820692777633667, 0.24244938790798187, -1.170475959777832, 1.067208170890808, -0.1686364859342575, 1.3600637912750244, 0.812681257724762, -0.002263353206217289, -0.37979891896247864, -0.9653412103652954, -1.2738571166992188, 0.18830065429210663, 2.0014889240264893, 2.242016553878784, 0.8194200396537781, -0.3995521366596222, 0.11809799075126648, -1.0257375240325928, 0.1958630383014679, 1.0231744050979614, -0.08406811207532883, -0.7359532713890076, 0.35951918363571167, -0.12638013064861298, -0.24639557301998138, -0.726830244064331, 1.4539259672164917, 0.47749999165534973, -0.028364907950162888, -1.9263206720352173, 0.2811618447303772, -0.20882964134216309, -0.3347802758216858, 1.6947227716445923, 0.9052469730377197, 0.8959938883781433, -0.9673714637756348, -0.19678033888339996, 0.7661756873130798, 0.7843418121337891, -0.11792730540037155, -1.7430217266082764, 1.309441328048706, 0.11607937514781952, 0.5808667540550232, -0.42641884088516235, 0.3328387141227722, -2.5058016777038574, 0.4584812521934509, 0.39457640051841736, 0.28238973021507263, -0.33561745285987854, -0.21939659118652344, -0.833486020565033, -0.3478662669658661, -0.9157987236976624, 0.8123764991760254, 0.9878158569335938, -0.3819883465766907, -1.3826439380645752, 0.38181090354919434, -0.6783491969108582, 0.18158209323883057, -1.1899142265319824, 0.2433209866285324, -0.35297635197639465, 0.39098992943763733, 0.6301499605178833, -0.05611829087138176, 0.9302265048027039, -0.05724726617336273, -1.7672193050384521, -0.2269570231437683, 0.2600974142551422, -0.47086089849472046, -0.3326047658920288, -0.5229357481002808, 0.18921653926372528, 1.160836100578308, -1.1355215311050415, -0.7230517864227295, 1.3626011610031128, 1.1382701396942139, -0.2695334851741791, -2.0516197681427, -1.1375582218170166, -0.9698330163955688, 0.9823732972145081, 0.5492860078811646, 1.9401445388793945, 0.17804796993732452, 0.5234114527702332, 0.08092544227838516, -0.2931528091430664, -0.4559359848499298, -0.4466361403465271, 0.8029927015304565, -1.1496448516845703, 0.791154682636261, -1.201806664466858, -0.11787691712379456, 0.6789433360099792, 0.2411700338125229, 0.5042451024055481, 0.25143128633499146, 0.8939381837844849, -1.0619251728057861, 0.7658842206001282, -0.573061466217041, 0.4661228656768799, 0.5762974619865417, 0.48036691546440125, 2.398899793624878, -0.43035006523132324, -0.42723608016967773, -0.3835670053958893, -0.0966557040810585, 0.8381685018539429, 1.0226414203643799, 0.07697395980358124, 0.7046387791633606, 0.098325714468956, 0.6763911843299866, -0.24026308953762054, -0.17723448574543, 0.3342551290988922, 0.9047608971595764, 1.3746827840805054, -1.2101725339889526, -0.9581463932991028, -0.7406882047653198, 0.268962562084198, -0.15268629789352417, -0.5292180776596069, -1.920449137687683, -0.08047974854707718, -1.959286093711853, 1.3717433214187622, -1.5333260297775269, -1.1491360664367676, -0.08697979152202606, 0.7622974514961243, -0.43601343035697937, 1.4110827445983887, 1.6954354047775269, -1.0850461721420288, -0.8526296615600586, -0.7954747676849365, -0.6667974591255188, -0.37305110692977905, 0.20644444227218628, 0.5915655493736267, 1.6283643245697021, -0.14129124581813812, -1.0716094970703125, 0.8485081195831299, 0.3027704954147339, -0.2125084400177002, 1.5054794549942017, -0.09947014600038528, -1.3120487928390503, -1.1476207971572876, 1.1831071376800537, -0.5634925365447998, 0.025322340428829193, 1.209451675415039, 1.6164602041244507, -0.5842701196670532, -1.4168277978897095, -1.7110114097595215, 0.6394097805023193, 1.1354931592941284, 0.8121371865272522, 0.4312763512134552, -1.081053376197815, 0.11851374059915543, 1.9509077072143555, 1.260549545288086, 0.10464516282081604, -1.3257083892822266, -0.7909702062606812, -0.9365396499633789, 0.37720879912376404, -1.1806049346923828, -0.19112439453601837, 0.6515341997146606, -0.3180253505706787, 0.762978196144104, -0.3476245105266571, -1.3016949892044067, -1.6048040390014648, 0.4633590579032898, 0.46974194049835205, 0.9826306700706482, 0.2041357010602951, 0.043402694165706635, 0.6475814580917358, 0.48774686455726624, -2.151947021484375, -0.06890644878149033, -0.4557236433029175, -1.2362053394317627, -0.1509179025888443, -1.0467331409454346, 1.0218311548233032, 1.0860764980316162, -1.1432567834854126, 0.7472706437110901, -2.3071751594543457, -0.06041533872485161, -0.6973950862884521, 0.13536588847637177, -0.6736600399017334, -1.9224909543991089, -0.4261334240436554, -0.40235209465026855, -0.6747071146965027, 1.7395212650299072, -0.663000762462616, 1.0376231670379639, -0.6448155641555786, -0.4513171911239624, -0.23989561200141907, -0.4358505606651306, 1.8364640474319458, -0.07068081200122833, 0.44347816705703735, 1.8340870141983032, 0.15985946357250214, 1.1091420650482178, 1.2526143789291382, 0.910757839679718, 0.8464482426643372, -0.02491104044020176, 0.8724640011787415, -1.0720843076705933, 1.0065780878067017, -0.1842627078294754, 2.383958101272583, 0.4395841658115387, -0.8507684469223022, -0.36804184317588806, 1.1307743787765503, -1.1397888660430908, 1.5231562852859497, -0.4367790222167969, 0.757314920425415, -0.12141481041908264, 1.5850731134414673, 0.6584046483039856, 0.4139045178890228, -0.4987398386001587, -1.0788395404815674, 0.23618854582309723, -0.4145885109901428, 0.42266109585762024, 0.7758591771125793, -0.3660653233528137, 1.1417149305343628, 0.9035656452178955, -0.758561372756958, 1.2365494966506958, -0.5001250505447388, -0.25485149025917053, -2.733701467514038, -2.8623523712158203, 0.43974804878234863, -0.06805554777383804, -1.0561566352844238, 0.08166555315256119, 0.9901110529899597, 1.0725799798965454, 0.7992321848869324, 0.2544039487838745, 1.8944414854049683, -1.1802771091461182, 1.2626688480377197, -1.6594834327697754, 0.5493480563163757, 1.0247780084609985, -0.05851038545370102, 0.17078952491283417, 1.6561864614486694, 1.2892006635665894, 1.3248454332351685, -1.0886235237121582, 1.5667269229888916, -0.2836121618747711, 0.2048460990190506, 0.9516554474830627, -0.9563331007957458, -0.5770044922828674, -0.24824930727481842, 0.8085756301879883, 1.384291410446167, 2.236612558364868, 0.41165435314178467, -1.8915046453475952, -1.2279672622680664, -0.6354721188545227, -1.0396908521652222, 0.5578834414482117, 0.11137112975120544, 1.9864165782928467, -0.21034705638885498, 0.6874715685844421, -0.2309582382440567, 0.08744042366743088, 1.436599850654602, 0.25633057951927185, -0.581882894039154, 1.1030749082565308, -0.4015626013278961, 0.29584914445877075, 0.16368024051189423, 0.04576105624437332, 0.8360214233398438, 1.015302300453186, -1.7386479377746582, -0.673492431640625, 0.8415953516960144, -0.950882077217102, 0.21841339766979218, 2.8105156421661377, -0.188929483294487, 0.5118470191955566, -0.9817119240760803, -0.8050519824028015, -0.6162000298500061, 0.2001938819885254, 0.8935873508453369, -0.5759350657463074, -2.3085343837738037, 0.8617227077484131, -0.3148643672466278, 0.3334607183933258, 0.0032044199761003256, -0.08234617859125137, -0.7837443947792053, 1.099145770072937, -0.7268046140670776, -1.19981849193573, -0.8291381001472473, -1.4575176239013672, -1.450114369392395, 0.3367530107498169, -0.12103306502103806, -0.07797703146934509, -0.7758302688598633, 0.42213112115859985, 0.4866780638694763, 0.15369023382663727, 2.2299704551696777, -0.2730315625667572, 0.031304530799388885, -0.297021746635437, -0.6073441505432129, -0.38952815532684326, 2.2826380729675293, 2.014138698577881, 0.31174638867378235, -0.4279756247997284, 0.08771847188472748, 0.1276022493839264, 1.1026333570480347, -0.04627833515405655, 1.2676212787628174, -0.6183916926383972, 1.2749253511428833, -0.7527382969856262, 1.5027060508728027, -0.12269458174705505, 0.3308614194393158, -0.15134918689727783, 0.5529530644416809, -0.4501400887966156, 0.06166212633252144, -0.21430538594722748, -0.322747141122818, 0.9003010988235474, -0.15996809303760529, 1.7902467250823975, 1.705706238746643, -0.7285295724868774, 0.6438372135162354, 0.6391129493713379, 0.7287857532501221, -0.5557434558868408, 0.16405446827411652, 1.281766414642334, -0.21203356981277466, -0.16800092160701752, -0.5925676226615906, 0.5021399259567261, 0.17421968281269073, 2.379917621612549, -1.7673635482788086, -0.7818093299865723, 0.11708921194076538, -0.48742562532424927, -0.9050791263580322, -0.4371696710586548, 0.41589799523353577, 1.2636361122131348, 1.2622030973434448, -0.08024317771196365, -1.2641221284866333, 0.4578385353088379, -0.3485446870326996, -1.6392412185668945, -1.0580462217330933, -1.553328275680542, 1.4538474082946777, -0.3049733340740204, 0.6207889914512634, 0.04885215312242508, 0.26071032881736755, -1.4065755605697632, -1.1706876754760742, 0.4030265510082245, 0.4373786747455597, 0.047547150403261185, -0.7977277040481567, 0.8543651103973389, -0.428218275308609, -1.5602920055389404, 0.9486942887306213, -0.24861657619476318, -0.9754326343536377, -0.28723642230033875, 1.8937934637069702, -1.3247849941253662, -1.5724736452102661, -0.5547701120376587, 1.24202561378479, 0.21896646916866302, -0.14532680809497833, -0.7898628115653992, 0.966609001159668, -0.8300527930259705, -1.5194764137268066, 0.4430980682373047, 0.4281441271305084, 1.1043901443481445, -0.11217433214187622, -0.5964781641960144, 0.6422514915466309, -1.954929232597351, 0.3292725682258606, 0.7558136582374573, -1.9131839275360107, -0.13777244091033936, -1.498016119003296, 0.3625915050506592, 0.6051648259162903, -1.4093971252441406, 0.48754292726516724, 0.14492957293987274, 1.2218297719955444, 2.6601498126983643, -1.3004724979400635, 0.6715738773345947, -0.6273899674415588, -0.46340110898017883, 0.12764613330364227, 0.8677123785018921, 0.4717097878456116, 0.7943179607391357, -0.8340728878974915, -0.05596011504530907, -1.0141607522964478, 1.9767670631408691, 0.3403824269771576, -0.44680655002593994, -0.795998752117157, 0.4299292266368866, -0.5699692368507385, -0.8928108811378479, -0.6484644412994385, -2.267144203186035, 1.1414519548416138, 1.3511664867401123, -2.1153790950775146, 1.5210176706314087, 0.7920196652412415, -1.7130979299545288, -1.5584797859191895, 0.42261821031570435, 0.8257256746292114, -0.44003206491470337, -1.780733585357666, 0.00013249987387098372, -0.6401810646057129, 0.7018095850944519, 0.1440807729959488, -1.0158733129501343, -1.8614964485168457, 0.021425122395157814, -0.8635812997817993, -0.1524105966091156, 0.4903922975063324, 0.1582413911819458, -0.7155711650848389, -1.2686303853988647, -0.11589404940605164, 1.4304561614990234, -0.11641047894954681, -0.9519645571708679, 0.620934009552002, 2.3428969383239746, -0.21113015711307526, 0.2286413013935089, -0.29623106122016907, -1.4915214776992798, 0.5777503848075867, -0.00804339163005352, 1.8765668869018555, -0.22337500751018524, 0.09709229320287704, -0.0674021765589714, 0.16270969808101654, -0.3719964921474457, -2.056586742401123, -0.08668313920497894, 0.4815244674682617, -1.8122999668121338, -0.7866328954696655, -0.8513604998588562, 0.07063524425029755, 1.5034257173538208, 0.23198576271533966, 0.7901788353919983, -1.0855604410171509, -0.48471811413764954, 1.1331207752227783, 1.0725526809692383, -0.7920857667922974, 0.6965479254722595, 0.14958418905735016, -1.635846495628357, -1.0944348573684692, 0.4059569239616394, -0.0047457958571612835, -1.3722425699234009, -0.8498486876487732, -0.4922694265842438, -1.310243010520935, -0.25168371200561523], [-0.693425178527832, 0.5919121503829956, -0.6117739677429199, -0.11225777119398117, -0.03788745403289795, -1.3471801280975342, 0.5650939345359802, -0.49735838174819946, 1.1424545049667358, -0.2060934454202652, -1.3768863677978516, -2.27309250831604, -0.8285885453224182, 0.7928940057754517, -0.6627109050750732, 0.4532155394554138, 0.65800541639328, 0.4428071975708008, -0.32962003350257874, 1.145828127861023, 0.3073126971721649, 1.1132704019546509, -1.3475502729415894, 0.13508917391300201, 0.6540469527244568, -1.6236473321914673, -0.4284678101539612, -1.969132661819458, 0.4152562916278839, 0.9147254824638367, -0.09163498133420944, 0.8726937174797058, 1.322053074836731, -0.2825263440608978, 0.5884289145469666, -0.794657826423645, -1.8960580825805664, -0.29110652208328247, 0.36442530155181885, -1.024025559425354, 0.3419013023376465, -1.211658239364624, 0.7900171279907227, -0.5449477434158325, 0.8794021010398865, -0.286457359790802, -1.1435014009475708, 0.06633365154266357, 0.5760002732276917, 0.28934478759765625, 0.9062806963920593, 0.5659927129745483, -1.296971321105957, 1.347368836402893, -0.10721172392368317, -0.4093582332134247, -1.0349438190460205, -0.8149823546409607, 2.1005115509033203, 0.5225122570991516, -0.7035643458366394, 1.3745505809783936, 0.07000815868377686, 0.6474642753601074, -0.8339788913726807, 1.3090299367904663, -0.5575087666511536, 1.6412076950073242, -2.848694086074829, -1.2942066192626953, 0.3603057861328125, 0.9683631658554077, 0.6196125745773315, -1.5408220291137695, 0.6958855986595154, -0.5711112022399902, -0.551893413066864, 0.8985888361930847, -0.3545267879962921, 0.6096373200416565, 2.0314395427703857, 1.6088745594024658, 1.238485336303711, 0.1301499754190445, -0.30632907152175903, 0.003396810730919242, 0.23553596436977386, 0.858878493309021, -0.3578336238861084, 0.4964373707771301, -0.42087066173553467, 0.22066354751586914, 1.210964322090149, 1.3380141258239746, -1.0083961486816406, 0.36405280232429504, 0.5745465159416199, -0.7233966588973999, 0.28493502736091614, 0.8801670074462891, -0.6621999740600586, -0.14954595267772675, -0.42110753059387207, 0.25462809205055237, 0.6832822561264038, -0.33341675996780396, 0.6122246980667114, -0.23943071067333221, -1.7188485860824585, -0.4342827796936035, -0.8485829830169678, 1.9937914609909058, -0.572809636592865, 0.9007014036178589, -0.1577325165271759, 0.11456869542598724, -1.258786678314209, -0.3914358615875244, -0.2501620650291443, 0.025615068152546883, -1.3627859354019165, 0.6423457264900208, -0.6712354421615601, -0.4035974442958832, 0.888631284236908, 1.4988857507705688, 0.19351889193058014, -1.1700876951217651, -0.7372012138366699, -0.5842720866203308, -1.1690385341644287, -1.440879464149475, -1.0482300519943237, 1.42205011844635, -0.20117132365703583, -0.47713327407836914, 0.0021999627351760864, 0.4987441897392273, -0.34590786695480347, -0.8109732866287231, -0.045695606619119644, -0.14046956598758698, -0.15450893342494965, -0.552570104598999, -0.6304709315299988, -1.1018842458724976, -0.19911085069179535, -0.39091041684150696, -1.703450322151184, 1.1077313423156738, -0.6158914566040039, 1.8210476636886597, -1.2984955310821533, 0.42808571457862854, -0.3565030097961426, -1.16801118850708, -1.274444818496704, -0.45196473598480225, -0.9201648831367493, -0.9004877805709839, 0.719009518623352, 0.36628904938697815, -0.6658035516738892, -0.9436968564987183, -0.8051637411117554, -0.21727456152439117, -0.3808021545410156, -0.2278687208890915, 0.3808983564376831, 2.466946840286255, -0.7193012833595276, -0.5373772382736206, 0.9656305313110352, -1.2164005041122437, 0.13396264612674713, 0.7305624485015869, -0.054533474147319794, -1.2229235172271729, -0.0125271026045084, 1.7833070755004883, 0.609118640422821, 0.23699744045734406, -0.06750904023647308, 0.4906155467033386, -0.7246604561805725, 0.024385487660765648, 0.047435179352760315, -0.22358770668506622, -0.976394534111023, -0.7845867872238159, -0.5817062258720398, 3.1015777587890625, -1.724253535270691, 0.17285187542438507, -0.0002952560898847878, -0.8491423726081848, -0.19961093366146088, -1.05994713306427, -1.2164554595947266, 0.47033724188804626, -1.8050353527069092, -1.1108980178833008, 0.004747574217617512, -0.9743325710296631, 0.7486760020256042, -0.8629741072654724, -0.10478145629167557, -2.9289753437042236, -0.8114065527915955, -0.8877254128456116, -1.4541205167770386, 0.8080337643623352, 1.8327192068099976, 0.3627217710018158, 0.1829092651605606, 0.2673932909965515, 0.12745164334774017, -0.7316204905509949, -0.5222388505935669, -0.1936049461364746, 0.2858717143535614, 0.8370610475540161, -0.5925626754760742, -0.7441480159759521, 0.5464704036712646, 1.627944827079773, 1.0385593175888062, 1.2663732767105103, 0.8351970911026001, 1.2817022800445557, 0.14865455031394958, 0.9630247354507446, -0.9155872464179993, -0.13454192876815796, 1.2815476655960083, 1.8572008609771729, -0.3047139644622803, 0.1470550149679184, -1.1160107851028442, -0.7391151785850525, 0.4066985547542572, -1.9467520713806152, -1.1580477952957153, -1.6626964807510376, 0.86650151014328, -1.909340739250183, -0.7687691450119019, -1.4447423219680786, -0.8387730717658997, 0.46911105513572693, 1.1982239484786987, 0.22463707625865936, 0.3960369825363159, 1.6717575788497925, -1.3892861604690552, 2.8496768474578857, 0.657132625579834, 0.651310384273529, 0.0680479109287262, -2.492940902709961, 0.9753591418266296, 0.01883010007441044, -0.4068571627140045, 0.9152806997299194, 0.6579788327217102, -0.5112168788909912, -0.34898075461387634, 1.358333945274353, -0.2719450294971466, 0.830331563949585, -1.3605613708496094, -0.18934977054595947, -1.9839493036270142, 0.754753828048706, 0.1944829374551773, 0.7523001432418823, 2.382809638977051, -1.5673397779464722, 0.22015663981437683, 0.9048863649368286, 0.004269774537533522, -0.0702672004699707, -0.1989535242319107, -0.39877164363861084, 0.11576804518699646, 0.5876049995422363, 1.5190987586975098, 0.5618496537208557, 0.03923257812857628, 0.06281254440546036, 0.6896139979362488, -0.009605931118130684, 0.43211907148361206, 0.026948826387524605, -0.6622941493988037, 1.4398159980773926, -1.1253794431686401, -1.4461967945098877, -1.9432450532913208, -0.9347282648086548, -0.7665472626686096, -0.4797540605068207, 0.3402497470378876, -0.2618933320045471, 0.004196117166429758, 0.5986430048942566, 1.0533963441848755, -0.6028320789337158, -1.2350467443466187, 0.07707688212394714, -0.5776525735855103, -0.8091863989830017, 0.9790257811546326, 1.813126564025879, 0.8094591498374939, -0.18049797415733337, 0.2814551293849945, -0.6076168417930603, 0.07716665416955948, -0.8182492256164551, 0.3366919457912445, 0.7125754952430725, -0.2181691974401474, -0.1842925101518631, -0.5053322315216064, 1.0300897359848022, -0.1581142544746399, 0.8402666449546814, 0.8933331370353699, -0.6779072880744934, -0.44409531354904175, -0.3319849669933319, 0.717812180519104, 0.873760998249054, -0.5311112999916077, 0.507479727268219, 0.01142018660902977, 0.26418372988700867, 0.1623922437429428, -0.433368057012558, -2.124854326248169, 0.8176007270812988, 0.7891624569892883, 0.28460147976875305, -0.8132007122039795, 1.3622239828109741, -0.4217260777950287, -1.1468067169189453, -0.24106226861476898, -0.17340809106826782, -1.8622416257858276, 0.41432610154151917, 0.5746874213218689, -2.4485511779785156, 0.8883306384086609, 0.10515309870243073, -0.33421769738197327, 1.0664596557617188, 0.19726309180259705, 0.6942546963691711, -0.4741465747356415, 0.02944858931005001, 0.7193674445152283, -0.2557603120803833, 0.688545823097229, -0.6081108450889587, 0.8411144614219666, -1.0925084352493286, 1.1505564451217651, -1.9866764545440674, 0.6701058149337769, -0.5508902668952942, -0.5272307395935059, -0.43856626749038696, -0.42283114790916443, 0.25809407234191895, 0.49105823040008545, -1.3703750371932983, 0.6456297636032104, -0.6372203230857849, -0.622880756855011, 1.6988847255706787, 1.4935321807861328, 0.4149719476699829, 0.9279356002807617, -0.7272672653198242, 2.2977168560028076, 0.31177040934562683, 0.5146063566207886, 0.45962104201316833, -0.33869022130966187, -0.35686245560646057, -0.5658905506134033, -0.5618687868118286, -1.673776388168335, 0.7017554640769958, -0.01573355495929718, -0.5452130436897278, 0.9104558825492859, -1.0831053256988525, -1.1284666061401367, 0.9719032049179077, -0.6663276553153992, -0.19960293173789978, -0.5790614485740662, -0.8428537249565125, 0.2346523255109787, -0.15215805172920227, 0.6782166361808777, -0.5085427761077881, 0.7447949051856995, -2.147101402282715, 0.6348564624786377, -1.05522620677948, 1.4498411417007446, 0.130998432636261, 0.47508758306503296, 0.2645946741104126, -0.8186801075935364, -1.6509257555007935, -0.22032447159290314, 0.05179746821522713, 1.1244858503341675, -0.8958064317703247, -0.10228399187326431, 0.34006258845329285, -0.7209744453430176, 1.0229582786560059, 0.39658066630363464, -0.4794357419013977, -0.027559731155633926, 0.3542156517505646, 1.5229661464691162, 1.1685419082641602, 0.7910262942314148, 0.613365888595581, 1.698753833770752, -0.7945778369903564, -0.2520028352737427, 0.17596480250358582, -0.48456865549087524, -0.6115642189979553, 0.8486226201057434, 1.2934900522232056, 0.4021742641925812, 1.0718640089035034, 1.0232056379318237, -0.10023275762796402, 1.8211098909378052, -1.525982141494751, 1.008828043937683, -2.2267425060272217, 0.35474273562431335, -1.142500400543213, -0.4295307397842407, 0.6675403118133545, 1.1606364250183105, 0.4998132288455963, -0.7949336767196655, -2.0511279106140137, 0.1546793431043625, 1.4795960187911987, -0.41060924530029297, -0.23106977343559265, -1.20743989944458, 0.07340537011623383, -0.25398239493370056, -1.9146867990493774, 0.814491868019104, -0.42461279034614563, -0.8039467930793762, -0.3603981137275696, 0.12359495460987091, 1.47234308719635, -0.6213369965553284, -1.2842187881469727, -0.6655930280685425, -1.2768877744674683, 1.1452542543411255, -1.031668782234192, 1.311287760734558, 1.4930075407028198, 1.7847976684570312, 0.32598602771759033, 0.549190878868103, -1.1781514883041382, -0.53226637840271, -3.1099565029144287, -0.49231699109077454, -0.7631162405014038, 1.1175408363342285, -1.1365303993225098, 1.4192733764648438, -0.058385659009218216, 3.521918296813965, 0.1176241859793663, -1.5252376794815063, -0.03072100132703781, 0.24127912521362305, -0.4031592309474945, 0.0526256188750267, -0.9999470710754395, -1.7103314399719238, -0.1345028579235077, 1.081077218055725, -2.0506105422973633, 1.1585559844970703, -0.04929574206471443, 2.2130234241485596, 2.2158994674682617, -0.3840164840221405, 0.8739645481109619, 0.5242064595222473, -0.20449009537696838, 0.8519986867904663, -1.6013529300689697, -0.5765775442123413, -1.5107288360595703, -0.51951664686203, -0.06278892606496811, -1.6724046468734741, -0.7981723546981812, 1.0402987003326416, 0.3382740914821625, -1.6409194469451904, -0.354514479637146, -0.015420185402035713, 1.0222512483596802, 0.5460180044174194, -0.07154063880443573, 0.26536718010902405, -2.6897287368774414, -1.0803526639938354, -0.7084893584251404, -0.17748241126537323, 1.840436339378357, -0.5634156465530396, -0.3737961947917938, 1.04725182056427, 2.3029134273529053, 1.856182336807251, 0.1302734762430191, -0.43308061361312866, 0.6560371518135071, -0.6584144830703735, -1.0280507802963257, -0.06985438615083694, 0.8035598993301392, -1.881289005279541, -0.4433710277080536, 0.9065321683883667, -1.3443551063537598, -2.411017417907715, 0.19025997817516327, 0.17610202729701996, 1.5870964527130127, -0.5399026274681091, 0.25743839144706726, -0.6320586204528809, -1.142055630683899, 2.127779006958008, 1.1686005592346191, -2.431504726409912, -1.9601393938064575, 0.8305512070655823, -0.4231400191783905, -0.36190560460090637, 0.6570599675178528, 0.6916736364364624, -0.34991902112960815, 1.6279231309890747, -0.22644786536693573, -0.21079353988170624, -0.2463330179452896, -0.8875256180763245, 0.041190024465322495, -0.31055763363838196, -0.9017623066902161, 1.6655607223510742, 0.6049087643623352, 0.05049009621143341, -0.13530421257019043, -0.8114399909973145, -0.5439240336418152, -0.11759751290082932, 0.9180327653884888, -0.951060950756073, 1.2303383350372314, 0.7267441153526306, -0.8963482975959778, 0.6952630877494812, -0.458710640668869, 1.1138453483581543, -0.021009553223848343, 0.758178174495697, -0.4986205995082855, -0.6026542782783508, -0.211223766207695, 0.11209593713283539, -1.2796205282211304, 0.2615593671798706, 0.7857206463813782, -0.19183683395385742, 1.4260528087615967, 0.0359509214758873, -0.5957001447677612, -0.5316836833953857, -1.0427221059799194, -0.37625443935394287, -0.42240917682647705, -0.42787793278694153, -0.6148535013198853, 2.7287864685058594, 1.5311702489852905, -0.7307816743850708, 1.2567533254623413, 0.636057436466217, 1.1801915168762207, 0.02211122028529644, 1.614562749862671, -0.5486249327659607, -1.1894899606704712, 0.20642925798892975, -1.8845092058181763, 0.6018304228782654, 0.759225606918335, -0.977577805519104, -2.3401811122894287, -1.660807490348816, -1.3402442932128906, 0.31150349974632263, -0.8419349193572998, 0.24792525172233582, -1.2113171815872192, 0.18535199761390686, -1.252051591873169, 1.6845782995224, 0.37650224566459656, -0.6930937767028809, -0.9697213768959045, 1.1675859689712524, -2.301459550857544, 0.6162235140800476, -0.2472289353609085, 0.011347255669534206, 0.34365367889404297, -0.47297853231430054, 1.4541758298873901, 0.12705391645431519, 0.11754528433084488, 1.2725028991699219, -1.2637733221054077, -1.1090983152389526, 1.736527681350708, 0.5277408361434937, 0.4457372725009918, 0.17840375006198883, -0.20073972642421722, 1.3788981437683105, 0.8268843293190002, -0.6826751232147217, -0.031498074531555176, 0.4854918122291565, 0.8878595232963562, -1.1825876235961914, -0.2241751253604889, -0.8465022444725037, -0.6288703680038452, -0.6214858889579773, 0.7366817593574524, -3.3496668338775635, 1.0147175788879395, 0.736393392086029, -0.36452388763427734, 1.526240587234497, 0.5538492202758789, 0.45420706272125244, -1.3289902210235596, 1.1753789186477661, 0.38889849185943604, 0.4078846573829651, 0.9782837629318237, -0.46444904804229736, 1.1138036251068115, 0.7999739050865173, -1.0241944789886475, -0.829912543296814, 0.9420835375785828, 1.0991075038909912, -1.2882744073867798, -0.39376360177993774, 0.6188276410102844, -0.29688361287117004, 0.5994896292686462, -0.06772913038730621, 0.06575847417116165, 0.2966984510421753, -1.0671128034591675, -0.5215169787406921, 0.8759751915931702, -0.1757659614086151, 2.171816110610962, 0.4277087450027466, -0.32907789945602417, -1.1655339002609253, 0.8577072620391846, -0.3275997042655945, 0.3734130561351776, -1.0138500928878784, 0.281362384557724, -0.7247790694236755, 1.4232128858566284, 0.2363620400428772, 2.5455734729766846, -0.9248918294906616, 1.0265058279037476, -0.8474509119987488, 0.11992210149765015, 1.0267961025238037, 1.163370966911316, -1.48969566822052, 0.10715867578983307, -0.646348774433136, -0.6237130761146545, -0.09261391311883926, 2.0074453353881836, -0.1438663750886917, -1.3668899536132812, 2.345515489578247, 0.29694312810897827, 0.7666093707084656, -0.27634090185165405, -1.5061854124069214, 1.0128153562545776, 0.46244242787361145, 1.0317243337631226, 1.2688931226730347, -0.5899732112884521, 0.31325390934944153, 3.011383295059204, -0.77482670545578, -1.1170140504837036, 1.0591734647750854, -0.0029093357734382153, -0.14975526928901672, 1.2892581224441528, -0.45980483293533325, 1.2028021812438965, 0.9249509572982788, -0.4268890619277954, -0.6683633327484131, 1.4309061765670776, -0.5568228960037231, 0.06232021749019623, -0.25565293431282043, 0.754846453666687, -0.09310348331928253, 2.0945844650268555, -0.40308433771133423, 0.7892765998840332, 2.044065475463867, -0.9318132400512695, -0.8682698011398315, 0.6961792707443237, 0.05078252777457237, 1.4731030464172363, -0.8710874319076538, -0.4024929404258728, -2.8107736110687256, -1.8531781435012817, 1.842856764793396, -0.600463330745697, -0.3882102072238922, 1.0838199853897095, -0.4510348439216614, 0.5913128852844238, 0.3619828224182129, -1.1404027938842773, 1.054461121559143, -1.186699628829956, -0.5162298083305359, 1.6070690155029297, 1.1183017492294312, 1.012399673461914, -0.8393916487693787, 0.45805710554122925, -0.6645371317863464, -0.9139789342880249, -0.3060818910598755, -2.2335028648376465, 0.017469970509409904, 0.13642854988574982, 0.862765371799469, 1.2259666919708252, -1.1997528076171875, 1.9785078763961792, 0.11843310296535492, -1.4237895011901855, 0.06442435085773468, -0.5623071193695068, 0.5372327566146851, 0.08742260187864304, -0.11349287629127502, 0.4967428743839264, 0.5522775650024414, -0.2012689709663391, -0.570763111114502, 1.0783048868179321, 1.7030856609344482, 0.4846706688404083, -0.07922287285327911, -0.9035722613334656, 0.25610968470573425, 1.438285231590271, 1.3039021492004395, -0.2526487112045288, -1.5005178451538086, 1.1213607788085938, 1.1237460374832153, 0.8236168622970581, 0.7876760363578796, -0.8555004000663757, -0.17215611040592194, 0.13825391232967377, 0.7560919523239136, 0.47274231910705566, -0.40243080258369446, 0.9199390411376953, 0.05392630025744438, -0.3741380572319031, 0.7799248695373535, -0.40293386578559875, 0.5096606612205505, -0.1351291537284851, -0.8643577098846436, 0.7856998443603516, -0.2437601387500763, 1.1396859884262085, 1.547478437423706, -0.2726484537124634, -0.5288947224617004, 0.5109905004501343, 0.4144272208213806, -0.12255465239286423, 0.5454137325286865, 1.651196837425232, 0.7953923344612122, -0.6321686506271362, 0.7622326612472534, 0.4106951355934143, -1.622890591621399, -1.1445460319519043, 0.8400434851646423, 1.6109771728515625, -0.5574830174446106, -0.6029356122016907, 0.8057612180709839, 0.6591214537620544, 1.1442238092422485, 1.097029685974121, 0.7168871164321899, 0.26807862520217896, -0.5125594735145569, 1.2429742813110352, -1.785838007926941, 0.8579639196395874, -1.570712685585022, -1.102476954460144, -0.42448511719703674, 0.3460089862346649, -1.9764549732208252, 0.08849577605724335, -1.102231740951538, -0.4070145785808563, 0.724996030330658, -0.5518372058868408, -0.3410535454750061, -1.5345419645309448, 1.2378143072128296, 0.6188350915908813, -0.2860308289527893, -0.19880107045173645, -0.690898060798645, 0.7199838161468506, -1.3499090671539307, 1.041297197341919, -1.085118055343628, -1.9198315143585205, -0.016743725165724754, 1.7165532112121582, -1.3852829933166504, 0.13540123403072357, 2.8753557205200195, 1.0650662183761597, 0.9687249660491943, 1.231428623199463, -0.2638038098812103, 0.3098442554473877, -1.301850438117981, -0.43988966941833496, 0.9953094720840454, -1.052426815032959, 1.1584062576293945, -0.5699727535247803, -0.9004884958267212, -0.9526488780975342, 0.44578033685684204, -0.018265964463353157, 1.3084731101989746, 2.150852680206299, -0.2881700098514557, 1.3118544816970825, -1.58307945728302, 1.0367101430892944, -0.9525657296180725, 0.09339487552642822, 0.1953263133764267, -0.12141413986682892, 1.2014598846435547, -0.9065714478492737, -1.0762035846710205, -1.8832992315292358, -0.2196476012468338, 0.15814344584941864, -0.7121734619140625, 1.0430582761764526, 0.1350439488887787, 0.04905613139271736, -0.31758391857147217, 0.7192139625549316, -0.09398238360881805, -1.8599326610565186, 0.3899848759174347, -0.9975859522819519, 0.9659003615379333, 1.8714778423309326, 1.3289133310317993, 1.1547081470489502, 1.2498825788497925, 1.843982219696045, 0.9957889318466187, -1.5416514873504639, -0.7794314026832581, 1.4473910331726074, -0.4917903542518616, 0.37054598331451416, 0.5010406970977783, -0.2504423260688782, -0.2991613745689392, 0.09130779653787613, -0.1737949103116989, 0.6149565577507019, 1.6032421588897705, 0.8456474542617798, -0.19394554197788239, -0.30355823040008545, -1.146722674369812, -1.4788267612457275, 0.04065489396452904, 0.0416092723608017, 0.14396116137504578, 0.1365707963705063, 1.0333993434906006, 0.588585615158081, 0.9428724646568298, -0.8744269609451294, -0.010897439904510975, 0.3957426846027374, -0.72372967004776, 1.0918329954147339, -1.1645687818527222, 0.9808610081672668, -1.7123476266860962, 0.15102727711200714, -1.8853683471679688, -0.08977610617876053, -1.0926363468170166, -2.44608998298645, -2.0647339820861816, -0.3727821707725525, -0.4149174094200134, 0.5461192727088928, -1.3361655473709106, 1.036023497581482, -0.3535863757133484, 0.8112459182739258, 0.7128779292106628, -0.0996769368648529, -0.7657169103622437, -0.8699227571487427, -0.44135987758636475, 0.34029799699783325, -0.6366508603096008, 1.1091862916946411, -2.4676198959350586, -0.5192715525627136, -1.121397614479065, -1.2889723777770996, -0.39658376574516296, 0.5281236171722412], [-1.237041711807251, 0.8484282493591309, -0.23307430744171143, -0.9286805987358093, 0.3068000078201294, -0.6551759839057922, 0.7710647583007812, -0.6383876800537109, 0.8561916351318359, -1.639724850654602, 1.96021568775177, 1.183301568031311, 0.5614513754844666, 0.13678742945194244, 0.7196149826049805, 0.9030342102050781, -1.0731755495071411, 0.5773744583129883, -1.9759291410446167, -0.6901370286941528, 1.5752214193344116, 0.2816286087036133, 1.4674729108810425, -0.769273579120636, -0.43817248940467834, -0.45497217774391174, 1.8885174989700317, -1.008212924003601, -1.3498327732086182, 1.2614282369613647, 0.6673303842544556, 0.9281889200210571, 1.229737401008606, -0.22607295215129852, 0.984228253364563, 0.09810764342546463, 0.13736797869205475, -1.0644773244857788, -1.1570018529891968, -0.20738756656646729, 0.7317639589309692, -0.7531260848045349, -1.505078911781311, -0.5380516052246094, 0.6613121628761292, 0.13285084068775177, -0.5310419201850891, -0.7101868987083435, -0.037531182169914246, -0.3525316119194031, 0.2755254805088043, -0.06697776168584824, 1.6469998359680176, 0.46530207991600037, -0.38125699758529663, 0.7193057537078857, -0.040516629815101624, -1.3126176595687866, 0.5862320065498352, 1.8784185647964478, -0.6272875070571899, -0.4922299087047577, 1.0170788764953613, -0.031867071986198425, 0.6149278283119202, 2.80424165725708, 1.3181504011154175, 0.176668182015419, 1.1524537801742554, 1.5584241151809692, -1.2550444602966309, -0.3289702534675598, 0.36606407165527344, 1.678949236869812, 0.6196120381355286, 0.8717981576919556, 0.02305024489760399, 1.1728250980377197, -0.04900996759533882, 0.6038970351219177, -1.700299859046936, 0.7431868314743042, -1.4769244194030762, 0.11230655014514923, -0.8080527782440186, 0.29521650075912476, -0.9492711424827576, -0.6749201416969299, 1.6297098398208618, 0.2084883749485016, -0.51720130443573, 0.3452933430671692, -0.5143465995788574, -0.5572416186332703, -1.0501694679260254, -0.6969552636146545, 0.41305503249168396, 0.5583214163780212, 0.7780821919441223, 1.865517020225525, 0.04017886891961098, 0.5327414274215698, -0.5973058938980103, 0.6502079963684082, -1.656911015510559, 0.6069290637969971, -1.1948957443237305, 1.6323477029800415, -0.15826980769634247, 0.3507053256034851, -0.7275423407554626, -0.5500805377960205, -0.16292811930179596, -0.31201839447021484, 0.26546600461006165, 0.3988555371761322, -0.3454746901988983, -1.0824980735778809, -1.7904924154281616, -0.49581047892570496, -1.268228530883789, 0.23767222464084625, 0.38786324858665466, -0.3582365810871124, -1.0925490856170654, -0.7984622716903687, 0.6883658766746521, -0.025856560096144676, 0.9858778715133667, -1.5551092624664307, 0.21164116263389587, -1.1277332305908203, -0.6130862832069397, -0.3291381895542145, -0.3646317422389984, -0.2559192478656769, 0.16836433112621307, 0.27990400791168213, -0.18259352445602417, 0.45351871848106384, 0.9565829038619995, 0.501240611076355, -0.1464962363243103, -1.124006986618042, 1.648972988128662, 0.43452975153923035, 1.7422407865524292, -1.015461802482605, 1.2233831882476807, 0.8716832995414734, -0.3682936728000641, -0.5910670757293701, 0.6503615975379944, 0.4255332350730896, 0.4597617983818054, -0.13141684234142303, 1.081404447555542, -0.19603291153907776, 1.1707119941711426, -1.016301155090332, -0.0953148603439331, -1.9640486240386963, -1.4987272024154663, 0.2806328237056732, 2.466740846633911, -0.4395390450954437, -1.493393063545227, 0.13013848662376404, 0.83921217918396, 1.0883245468139648, -0.5879442691802979, -0.8558039665222168, -0.27276965975761414, 0.45678871870040894, -1.8222367763519287, 0.49968022108078003, 1.2875670194625854, 0.4158393144607544, -1.446639060974121, -0.05998186767101288, -0.7924738526344299, 0.6931177973747253, 0.749122679233551, -1.7873468399047852, 0.7351135015487671, 0.41518348455429077, 2.419816732406616, -0.22455185651779175, -0.2026335895061493, -1.3493459224700928, 0.7683914303779602, 0.02868521958589554, -0.7034425735473633, -2.929898738861084, -0.17799024283885956, -0.03637843206524849, 0.11107734590768814, -0.7250648140907288, -0.27381742000579834, 1.6741067171096802, 1.0176042318344116, 0.16140510141849518, 0.9253549575805664, -0.7281915545463562, -0.09262151271104813, 0.9450339674949646, 0.7445685863494873, -1.306220293045044, -0.31090718507766724, 1.7919962406158447, -0.6547618508338928, -0.3096170425415039, -1.730627417564392, -0.9288948178291321, 2.2362184524536133, -1.4821172952651978, 0.23411382734775543, -0.0878056138753891, 1.2134348154067993, -1.660820484161377, 0.04264688119292259, 1.2913700342178345, -0.3920312225818634, -0.47774630784988403, -0.5374365448951721, 0.06518633663654327, -1.384684443473816, -3.496260166168213, -0.7690331339836121, 0.5277993679046631, 1.0062390565872192, 0.6082205772399902, 0.03307899087667465, 0.14073564112186432, -0.7840307950973511, 0.09954606741666794, -0.21056528389453888, -1.2785327434539795, 0.15680339932441711, -0.2796925902366638, -0.38034018874168396, 2.2129967212677, 0.09913872182369232, 0.07883652299642563, 0.06259412318468094, 0.07447269558906555, -1.9307712316513062, -0.37717974185943604, 0.4335205852985382, -0.7407064437866211, -1.313647985458374, 0.68076092004776, -0.3078349530696869, 0.3674231469631195, -0.02500663883984089, 0.8943438529968262, 0.21779808402061462, 1.0846869945526123, 0.5152835845947266, 0.40633416175842285, -0.3887312710285187, -0.21519847214221954, -0.4568385183811188, -1.5588613748550415, 0.7477133870124817, 0.6677670478820801, 0.8747860193252563, -0.9774909019470215, -0.21403859555721283, 1.9605563879013062, -0.091467946767807, 0.2646678388118744, 0.5760613679885864, -1.0572530031204224, -1.2174103260040283, 1.6986078023910522, 0.9195971488952637, 1.3295927047729492, -0.19713371992111206, 0.8915984630584717, -0.9909278154373169, 0.21679756045341492, -0.3178688883781433, -0.35583946108818054, -0.704721987247467, -0.748412013053894, 0.2514357566833496, 0.9138506054878235, 0.09404309093952179, -1.4819179773330688, -0.4462866187095642, 0.05464805290102959, -0.22000831365585327, -0.2792285680770874, -3.466796398162842, 0.7470661997795105, -0.9790594577789307, -0.7567026615142822, -1.4135255813598633, -0.5722705125808716, -0.018245244398713112, 0.45640915632247925, 0.2806006371974945, -0.99602872133255, -1.7718137502670288, 0.1203334704041481, 0.2872827649116516, 1.787272334098816, -0.1939268410205841, 1.261682152748108, 0.041781265288591385, 0.7638722658157349, -0.32236143946647644, -0.8975042104721069, 0.21829821169376373, -0.2669309377670288, -0.2932661175727844, -0.22075128555297852, -0.16632668673992157, 1.162097454071045, 0.05823158472776413, 0.3895234763622284, -0.9571852087974548, 0.44570061564445496, 0.17553146183490753, 0.3752443492412567, 1.3505563735961914, -1.5253404378890991, 1.3323016166687012, 0.7088481187820435, -2.013120412826538, -0.3222418427467346, 1.3319681882858276, 0.06764322519302368, 2.130876064300537, -0.11131004244089127, 1.3573521375656128, -0.0944746881723404, -1.1670608520507812, 0.6011883020401001, 0.8093201518058777, 1.1423728466033936, -1.2714215517044067, -1.8262450695037842, -0.3238063454627991, -0.4691716730594635, -1.6860089302062988, -0.15235675871372223, -1.0927013158798218, 0.9822565913200378, -2.2853450775146484, 0.4266875088214874, 1.756367564201355, -0.9991284608840942, -0.9365241527557373, 1.1832014322280884, -0.8689495921134949, -0.0760950818657875, -1.4615936279296875, -0.8790948390960693, 1.4727973937988281, 1.2758084535598755, -0.3946038484573364, -0.3347870111465454, -0.1943228840827942, -0.7760705351829529, -0.5975964069366455, -0.30174508690834045, 0.30565711855888367, -0.7782081961631775, -0.5030068159103394, 0.15781764686107635, 0.35784393548965454, 0.7690703868865967, 1.389520287513733, 0.9420666694641113, 0.9628016352653503, -0.838643491268158, -0.09611383825540543, 1.9609076976776123, -1.218754529953003, -0.5411294102668762, -2.0084006786346436, 0.0841398760676384, -0.887505054473877, -0.8737598061561584, 0.6071135997772217, -0.6517862677574158, -1.161520004272461, 2.543046236038208, -1.0488684177398682, 1.9662681818008423, -0.38065463304519653, 0.48359227180480957, -0.48110097646713257, 1.3635432720184326, 0.602839469909668, 0.0950094684958458, -0.5742511749267578, -1.1900420188903809, -1.2443673610687256, 0.181365504860878, -1.648339867591858, 0.004423023201525211, 1.1229820251464844, 1.4069348573684692, 0.545140266418457, 0.24247722327709198, 1.2815221548080444, 0.31018710136413574, 0.5503737926483154, 0.8301011323928833, -0.9546433091163635, -0.7564924359321594, -0.2770632803440094, 0.059778325259685516, -0.2675783336162567, -0.5661183595657349, -0.2886998653411865, 0.41543343663215637, -0.18345077335834503, -1.2833961248397827, 2.281153678894043, 0.12800747156143188, 0.8412688970565796, -0.28234902024269104, 0.21726194024085999, 0.952416181564331, 0.89955073595047, 0.23678210377693176, -0.5069833993911743, 1.8814294338226318, 0.49217289686203003, -0.008008807897567749, 0.5216599106788635, 0.5408170223236084, 0.636635959148407, 0.9053897261619568, 2.162041425704956, 1.858475685119629, 0.09945753961801529, -0.3510895371437073, 1.629036545753479, 0.9869180917739868, 0.2212391495704651, -0.040386296808719635, -0.4075474441051483, 0.057000771164894104, 1.6706770658493042, 0.23320415616035461, -0.9035442471504211, -0.469074010848999, 0.685977578163147, -1.5687154531478882, 0.34885576367378235, -0.0052586211822927, -1.0437536239624023, -0.19008924067020416, 1.484858512878418, -1.8315945863723755, 0.969262957572937, -0.24252307415008545, -0.19845110177993774, -0.0014512947527691722, 1.515482783317566, -0.6065644025802612, 0.972578227519989, -0.007381850387901068, 1.1486433744430542, -0.22596099972724915, 1.337531566619873, -0.228685200214386, 0.35260024666786194, 0.47423478960990906, 1.0651837587356567, 0.4649518132209778, 0.49009114503860474, -1.7431925535202026, 1.0060386657714844, 0.47284337878227234, 0.00845061894506216, 0.06799885630607605, 0.9598081111907959, -0.29240521788597107, -1.744793176651001, -1.186444878578186, 0.5830721855163574, -0.12976935505867004, 0.9663917422294617, -1.1218429803848267, 0.597790002822876, 1.5700716972351074, -0.37159013748168945, -0.4109995365142822, -0.052840009331703186, -1.5444791316986084, 0.8070469498634338, 1.6536242961883545, -0.19869108498096466, -2.0545716285705566, -1.1655369997024536, -0.35913383960723877, 0.8244971036911011, 1.2212960720062256, -0.8836678862571716, 1.1765364408493042, -0.03931313753128052, -0.32025283575057983, 0.8617145419120789, -0.3131142556667328, -1.3171879053115845, -1.141190767288208, 2.1462018489837646, -1.3474665880203247, -0.9730631113052368, -0.5920567512512207, 0.8461184501647949, 1.2268437147140503, -0.04567619413137436, -1.3510990142822266, -1.1307066679000854, 0.07267753779888153, 0.32141631841659546, 0.29527896642684937, 0.6934685111045837, 2.1572723388671875, -0.5091668367385864, -0.5477373003959656, -0.23028968274593353, -0.4927707314491272, 0.1686108410358429, -0.6132816076278687, 1.5939003229141235, -0.5563700795173645, -0.5551502108573914, 1.0761576890945435, -0.22176681458950043, 1.5502301454544067, 2.028722047805786, -1.4596819877624512, 0.05020488426089287, 0.07751186937093735, 0.23184531927108765, -0.6671537160873413, -0.34176942706108093, -3.0102922916412354, 0.4949444830417633, -0.8434358239173889, 0.6295990347862244, 0.8433766961097717, 0.8824828267097473, 0.9601649045944214, -0.27325522899627686, 0.18596066534519196, 0.47344082593917847, -0.2191801220178604, 0.8063107132911682, -0.3452737629413605, 0.9880852103233337, 1.9355801343917847, -0.4655628800392151, 0.4996972978115082, 0.5821115970611572, 1.3000308275222778, 0.058159034699201584, -1.7147828340530396, 2.2800188064575195, -1.156469464302063, 0.07121630012989044, 1.1386574506759644, -0.5283873081207275, 0.16797253489494324, -0.6431238055229187, 0.5178541541099548, 0.8698909282684326, 1.0033388137817383, 1.4380016326904297, 0.021196018904447556, 0.35693976283073425, -0.568819522857666, 1.1091862916946411, -1.2335580587387085, -0.4304695427417755, 1.5891591310501099, 0.0801258310675621, 1.3434057235717773, -1.0192126035690308, 1.2868101596832275, 1.6542500257492065, -0.4621182978153229, -1.8569221496582031, 0.8328853249549866, 1.1988319158554077, 2.370882511138916, 0.2340613603591919, 0.7754551768302917, 0.6345793008804321, -0.1860073208808899, -0.16213677823543549, -1.0911091566085815, -0.40065473318099976, 0.9856831431388855, 0.4150044023990631, 0.11251340806484222, 0.07650173455476761, -0.631025493144989, 0.41863247752189636, 1.287172794342041, -0.4288170635700226, -0.8961211442947388, 0.18803147971630096, 1.768080234527588, -0.8959894180297852, -1.220629334449768, -1.0617258548736572, 0.39357790350914, -0.1889699399471283, -0.8103271722793579, 0.47916486859321594, 0.15060903131961823, -0.896003246307373, -1.7702006101608276, 0.03516334667801857, 1.0247089862823486, 2.908222198486328, -0.1248917207121849, 0.4129137396812439, 0.49541762471199036, -1.7021585702896118, -0.7315893173217773, 1.2463725805282593, -0.29814958572387695, 1.2701387405395508, -0.7438411116600037, -0.290655255317688, 0.9771563410758972, -0.961818277835846, 1.1994236707687378, 0.904730498790741, 1.473114013671875, -1.2409634590148926, 1.2519607543945312, 0.3596469461917877, 0.3608818054199219, 0.10849212110042572, 0.003834490431472659, 1.5586044788360596, -0.36348026990890503, -0.5662844181060791, -0.810868501663208, -1.576087474822998, -0.3575403690338135, -0.06328443437814713, 0.3570476472377777, -0.5016034245491028, -0.5644994378089905, -1.5576918125152588, 0.04881075397133827, 0.7092387080192566, -1.4670315980911255, 1.6642524003982544, 0.2946833074092865, 0.07218910753726959, -1.1109753847122192, -0.3478696644306183, -0.545703649520874, 0.32770824432373047, 0.34988299012184143, -0.13621973991394043, -0.05936867743730545, 0.2600189745426178, 1.4911863803863525, -2.1441919803619385, -1.359275460243225, 1.4234693050384521, -1.3313063383102417, -0.02075917087495327, -0.4263707995414734, -0.7018499374389648, 1.2049412727355957, -0.3277019262313843, 0.5722854733467102, 0.577444851398468, 0.3529888689517975, 0.0684712752699852, -0.01139048021286726, -0.4526806175708771, -0.634171187877655, 0.266492635011673, -0.7034696936607361, -0.4602493941783905, 1.5819921493530273, -0.7207537293434143, -0.5183536410331726, -0.5271185040473938, -0.23711836338043213, -0.23229002952575684, 2.3030784130096436, 0.6138379573822021, 0.287747323513031, -0.9470562934875488, -0.5901179909706116, 0.970256507396698, -0.11247174441814423, 0.6806924343109131, -0.5734123587608337, -0.3794127404689789, -1.5522968769073486, 1.30691659450531, -1.5497241020202637, -0.21266353130340576, -0.16916343569755554, 2.2507236003875732, 0.6136977672576904, 0.9587455987930298, 0.26890701055526733, 0.43453308939933777, -0.15102024376392365, 0.5671194791793823, -0.30067142844200134, -0.5825440287590027, -1.1138646602630615, 2.3964717388153076, 0.10217304527759552, -1.1138702630996704, -2.106239080429077, -1.582531452178955, 1.0614103078842163, 0.9306592345237732, -1.2051202058792114, 1.2599906921386719, -2.001148223876953, 0.021862879395484924, -0.04149079695343971, -0.019146643579006195, 0.3587420880794525, 0.1410841941833496, -1.216537594795227, -1.0913763046264648, -0.1968226581811905, -0.07979070395231247, -0.9889940619468689, -1.499435544013977, 0.7455164194107056, 0.10885294526815414, 0.619478702545166, -0.19946499168872833, 0.1339806765317917, 1.8780522346496582, 0.9182616472244263, 0.9907472133636475, -0.028909455984830856, 0.182697594165802, 1.3553780317306519, -0.21806122362613678, 1.454803466796875, 1.8998942375183105, -0.378193736076355, 0.87222820520401, -0.4581366777420044, -1.0855817794799805, -2.152334451675415, -1.4850387573242188, -1.7455785274505615, -0.5461640954017639, 1.659268856048584, -0.2959061563014984, -0.08102064579725266, -1.406431794166565, 0.5292604565620422, -1.5698778629302979, -0.5661105513572693, 0.4358327090740204, 3.5443801879882812, -0.16928772628307343, 2.0540432929992676, 1.1654736995697021, 0.7188683748245239, 0.2542235255241394, 0.29789602756500244, -1.336382508277893, 1.785373568534851, 1.0023667812347412, -1.9860669374465942, 0.044837869703769684, 0.014641626738011837, -0.49629542231559753, -0.5138493180274963, -0.715763509273529, 0.3729694187641144, -0.5407130718231201, 0.8188965320587158, -0.3054671883583069, 0.7456780076026917, -2.0027883052825928, 0.5829315185546875, -1.7174288034439087, 0.27822333574295044, -0.18158721923828125, -2.781745672225952, 0.8868075609207153, 1.4360421895980835, 0.04157887399196625, 0.47320082783699036, 0.6425594091415405, 0.21521611511707306, 0.5338075757026672, 0.8739027380943298, 0.3085915744304657, -1.694964051246643, 0.019836116582155228, 1.727713704109192, 1.2365334033966064, -0.1650741547346115, -0.13291887938976288, 0.14785577356815338, -0.09900442510843277, 0.21678408980369568, 0.035631466656923294, 1.9115384817123413, 0.5539740324020386, -0.5775052309036255, -0.3293582797050476, -1.0507173538208008, 0.2489190399646759, 0.2246485948562622, 0.7434945106506348, -0.6235235929489136, 0.2634403705596924, 0.35355526208877563, 0.4055754840373993, -0.3798125684261322, -0.7234737277030945, -0.6455622911453247, -0.6684970259666443, -0.8427685499191284, 0.4943118393421173, 1.6702250242233276, 1.4669077396392822, -1.7631956338882446, 2.3427915573120117, 0.8589414954185486, 0.8487080335617065, 0.6863953471183777, 0.2468348890542984, 1.1376863718032837, 0.8578228950500488, -0.33851224184036255, -0.4351176917552948, 0.34471848607063293, 0.21418926119804382, 1.2739182710647583, -0.19796150922775269, 1.399574637413025, 1.1677016019821167, -1.2766804695129395, 1.8038197755813599, -0.02421436458826065, -0.6723070740699768, -0.6587764024734497, -1.2313919067382812, -0.569761335849762, -1.0591111183166504, -1.2726662158966064, 0.7119001150131226, 1.2267669439315796, -0.35496413707733154, 0.4807024300098419, 0.12590742111206055, 0.5764821171760559, -0.8578907251358032, 0.2589041590690613, 1.2962374687194824, -0.6627874970436096, -1.5163363218307495, 2.713752269744873, -0.21472975611686707, -2.6687071323394775, -0.28386303782463074, -0.676514208316803, 0.3380332589149475, -1.0093090534210205, 0.53587806224823, 0.6389074921607971, 1.4624358415603638, -2.0794942378997803, 1.055036187171936, -0.6112183332443237, 1.931020736694336, 0.4178294241428375, 0.6710572242736816, -0.6141006946563721, 0.02711614780128002, -0.07440409809350967, -0.026264460757374763, 1.5256787538528442, 0.7693872451782227, 0.7078189849853516, 0.8915821313858032, 0.4573550820350647, -0.2353401631116867, -0.1765126883983612, 1.0253342390060425, -0.24830247461795807, -0.11923888325691223, -1.6945563554763794, 0.8407185077667236, -0.6514225602149963, 0.5653488636016846, -0.4846995174884796, -0.38551947474479675, 1.531594157218933, -0.40981313586235046, -0.4048132300376892, -0.41978874802589417, -0.38264986872673035, 0.227134570479393, 0.2821407914161682, -0.8724966049194336, -1.0683976411819458, 0.8115451335906982, 0.8968372344970703, -0.3885732591152191, 0.22325433790683746, 0.8839064240455627, -1.3936951160430908, -0.9592041373252869, 0.6345769166946411, -0.40720584988594055, 0.2667911648750305, -0.6230276823043823, 0.24727104604244232, 0.41024190187454224, 2.154515027999878, -0.2761657238006592, -0.4551424980163574, 0.016033316031098366, 0.8478624820709229, -1.8105368614196777, -0.39987924695014954, 0.8753036260604858, 0.7986029982566833, 0.027895528823137283, -0.13023178279399872, 0.1709756702184677, 1.6753668785095215, 0.36145082116127014, -1.159011721611023, -2.3296027183532715, 0.7231185436248779, -0.7446311712265015, 0.011351334862411022, -1.287927508354187, -0.3838077783584595, -0.44534116983413696, -0.2017904669046402, 1.372642993927002, -0.016749663278460503, -1.1295949220657349, -0.6753684282302856, -0.6308547258377075, 0.3575294017791748, 1.0810997486114502, 0.9986565113067627, -1.1777706146240234, -0.00922086276113987, -1.0483351945877075, -0.5038732290267944, -0.021088991314172745, -0.03256446495652199, 0.13916736841201782, -0.9347591400146484, -0.9057446122169495, 0.26161691546440125, -1.7140145301818848, 1.1728724241256714, 0.33167192339897156, 0.837246298789978, -0.7820025682449341, 0.9886569380760193, 0.5059622526168823, -2.7700259685516357, -0.15459044277668, -1.7294286489486694, -0.30488431453704834, 1.0490144491195679, 0.7542726993560791, 1.0793005228042603, -0.28667762875556946, 0.43156111240386963, 0.05505887418985367, -1.3342076539993286, -0.8824170827865601, -0.6345649361610413, 0.7656874656677246, 1.2639837265014648, 0.7094064950942993, 1.9525251388549805, -2.0987637042999268, 0.45059776306152344], [0.4235938489437103, -0.6846623420715332, -0.06762361526489258, 3.3936922550201416, -1.0614880323410034, 0.3520011305809021, 0.23659925162792206, -0.8186730146408081, -0.5745865106582642, -0.10209416598081589, 1.355465292930603, 0.46267539262771606, 0.45083677768707275, 1.0837589502334595, 2.0681521892547607, 1.8910155296325684, 1.6416500806808472, 0.4421814978122711, -0.24261657893657684, -1.5824369192123413, -0.5345169305801392, 0.6287454962730408, 0.711725652217865, 0.510468065738678, -0.42104172706604004, -0.3516930639743805, 0.12028439342975616, 0.5071317553520203, -0.6082524061203003, -1.441164255142212, -1.101534128189087, 0.45615386962890625, 1.151098608970642, 0.03299877420067787, -0.3434128761291504, 0.7883898019790649, -0.4565533399581909, 1.1027644872665405, -0.21579593420028687, -0.7826938629150391, -0.1621427685022354, -0.5772700905799866, 1.0349353551864624, -2.6559207439422607, -1.6232459545135498, 0.5065938234329224, 2.022043228149414, 0.5881931781768799, 1.1553194522857666, 1.0086299180984497, 0.7676319479942322, 0.645656943321228, 0.17267455160617828, -0.24585163593292236, -0.19933630526065826, 0.5962271094322205, -0.09101415425539017, 0.3019363284111023, 0.1449098140001297, -0.42105376720428467, 0.5705300569534302, 0.5697742700576782, 0.19058693945407867, 0.04701763391494751, -2.277252197265625, 0.01332870777696371, -0.9219229817390442, -1.468582034111023, 0.45806047320365906, 0.9481378197669983, -1.1418594121932983, 1.226356029510498, -0.27226194739341736, 1.7041212320327759, -1.2398542165756226, 1.029716968536377, 0.7999157309532166, -0.32628974318504333, -0.5407080054283142, -0.31162482500076294, -0.4306469261646271, 0.07758671045303345, -1.7173001766204834, -0.7496635317802429, -0.9474748373031616, -0.24306343495845795, -1.674454927444458, 1.7378283739089966, -0.7039041519165039, -1.106644630432129, -1.6018790006637573, 0.33980467915534973, -0.5772431492805481, 0.10497701168060303, -1.2872023582458496, 0.2819382846355438, 1.2887170314788818, 0.365136981010437, 0.16546674072742462, 0.30154716968536377, -0.8641952872276306, -0.10119631141424179, -1.455370306968689, 0.18076831102371216, -0.321992427110672, -0.19680580496788025, -1.148301124572754, 1.3537712097167969, -0.8710697889328003, -0.11873531341552734, -1.6489815711975098, -0.8714768886566162, -0.02405409887433052, -1.477608323097229, -1.0516248941421509, 1.6402839422225952, -0.31782230734825134, -0.3175380825996399, -0.40208280086517334, -1.6995396614074707, 0.7688528895378113, -0.09698818624019623, -0.9975460767745972, -0.2661805748939514, 1.3626517057418823, 1.4137648344039917, -0.532724142074585, -0.09597761183977127, -0.8200543522834778, 0.4213072657585144, -0.38394084572792053, -0.7449237704277039, 0.7471106052398682, -2.2262086868286133, -0.15834437310695648, -0.4485786259174347, 0.28313377499580383, 0.07674699276685715, 0.5060017108917236, -1.5176236629486084, 2.0273475646972656, 0.4638568162918091, 1.299033284187317, -1.7987771034240723, -1.0069210529327393, -0.8484635353088379, -1.3901551961898804, 0.29379498958587646, 0.4726759195327759, 1.7227240800857544, 0.18103544414043427, -0.2824610769748688, 2.301276683807373, -0.6955165266990662, -0.11322037130594254, -1.0397768020629883, -0.8884899616241455, 0.9661595821380615, -0.12811413407325745, 0.6866628527641296, 0.469731867313385, -1.7246531248092651, 1.021744966506958, -1.3123209476470947, -0.19565078616142273, -0.9549009799957275, -0.9785279035568237, -0.8268875479698181, -1.4763648509979248, 1.8942102193832397, 2.208184242248535, -0.780726969242096, 1.3301317691802979, 0.36565184593200684, -0.870581328868866, 1.4972050189971924, -0.3365922272205353, -0.22881917655467987, -0.6375066041946411, -1.416542649269104, -0.4450560510158539, 0.3561249375343323, -0.5126977562904358, 0.9748092889785767, -0.4748722016811371, -1.5051385164260864, -0.4646539092063904, -1.718629240989685, -0.553155779838562, -0.4699830114841461, -1.6178569793701172, -0.3077073395252228, -0.2668578624725342, 0.11187203228473663, -0.9336937069892883, -0.5807600617408752, -0.7638578414916992, 0.07244992256164551, 1.974683165550232, 0.5836688280105591, -0.21940258145332336, -0.5762527585029602, 0.46102726459503174, 0.6707415580749512, 0.4015292525291443, -0.5802730917930603, 0.1398746222257614, -0.9491099119186401, 1.9988832473754883, 0.014862864278256893, -0.5442832112312317, -1.1625404357910156, -0.6399421095848083, -1.0041897296905518, -0.5848820805549622, -0.13851988315582275, -0.8214466571807861, 0.6323405504226685, 0.11478125303983688, 0.43094727396965027, -0.8048496842384338, 0.29147571325302124, -0.06639190763235092, -0.06852447986602783, -0.11720406264066696, -1.2387149333953857, 1.1570745706558228, 0.7602059245109558, -0.7035847306251526, 0.9640079140663147, -1.3821914196014404, -0.12522654235363007, -2.824432134628296, -1.073001503944397, -0.5096872448921204, 0.7719923257827759, 1.3199968338012695, 0.27197977900505066, 0.8593061566352844, 0.4713704288005829, 0.47382763028144836, -0.8565893173217773, -1.5255824327468872, 0.549703061580658, 0.8443737626075745, -0.6187816262245178, -0.7652687430381775, 0.32796838879585266, 0.22666259109973907, 0.6817483305931091, 0.16411124169826508, 0.8023942708969116, -1.7803986072540283, -0.3722871243953705, 0.9630342721939087, 0.2129993736743927, -0.7479667663574219, -1.2187737226486206, -0.0994122177362442, 0.8135668039321899, 0.7516228556632996, 0.7554455995559692, -0.34308746457099915, -0.8832838535308838, -0.6511468291282654, 1.1373754739761353, -0.6712806224822998, 0.9069393873214722, -0.027573758736252785, -0.5446750521659851, 0.33854493498802185, 1.4576356410980225, -1.0577259063720703, 0.940405011177063, -0.8408106565475464, 1.020538568496704, -0.11333723366260529, 0.8163458704948425, -0.06931018829345703, 0.42488008737564087, -0.8890851140022278, 1.6109189987182617, -0.5195376873016357, -0.7641413807868958, -0.7529333233833313, -1.1429426670074463, -1.2871755361557007, -0.9850797057151794, -1.092230200767517, 0.7890753746032715, -0.28877341747283936, -1.6267383098602295, 0.9668870568275452, 0.8873592615127563, 1.2708370685577393, 0.6790380477905273, 0.8260889649391174, -1.3705768585205078, -0.2418726533651352, 0.5635843276977539, 0.5640628933906555, -0.5530928373336792, 1.0316153764724731, -1.5457115173339844, 0.6311298608779907, -0.08374396711587906, -0.019972244277596474, -1.3482965230941772, 0.5884845852851868, 0.9942965507507324, -0.9897260069847107, -0.1492219716310501, -0.8278481364250183, 1.140945553779602, 0.9967688322067261, 1.3125450611114502, 0.9638513326644897, 0.7420511841773987, 0.037945497781038284, -1.736716389656067, 0.7606627941131592, 1.7308131456375122, -0.6747829914093018, -0.25682929158210754, 1.4010975360870361, -0.567716658115387, -1.3410377502441406, -0.4249134063720703, 0.4819020628929138, 0.034949690103530884, 1.0651663541793823, 0.4042389690876007, 0.37927010655403137, 1.1905899047851562, -0.2365962564945221, 0.07976110279560089, 1.3763800859451294, -0.10976997762918472, 1.954688549041748, 0.5540859699249268, 0.6801022887229919, -0.8909775018692017, -0.5844681859016418, 1.0438363552093506, -0.35524678230285645, 0.19630365073680878, -0.8748643398284912, 0.6170716285705566, -0.9793084859848022, -0.9612802863121033, -0.23945845663547516, -2.1270225048065186, 0.5041720867156982, 0.345685213804245, 0.5157670378684998, 0.004284022841602564, -0.5665146708488464, -0.26509249210357666, -1.279006838798523, 0.10009600222110748, -0.3725176453590393, 0.08281869441270828, 0.7473753690719604, 0.21063446998596191, -0.31694090366363525, -0.09324008226394653, -0.10748881846666336, 0.10563158243894577, 0.11600862443447113, -0.3318495750427246, -0.7866650223731995, 1.1241555213928223, 0.8179717659950256, 0.2478022426366806, 0.22998565435409546, -0.9364892840385437, 0.28774550557136536, -0.37119072675704956, 0.8099977374076843, 1.0746028423309326, 0.6763501763343811, -0.2935558259487152, 0.6051773428916931, -0.10265851765871048, 0.21223928034305573, 1.4999905824661255, 0.013406983576714993, -0.48976799845695496, 0.2272409349679947, 0.09884490072727203, -0.1969635784626007, -0.3053489327430725, 0.808112382888794, -0.9323298335075378, -0.6519550085067749, -0.3746841251850128, 0.22599081695079803, -2.5718588829040527, 0.24451787769794464, -1.5448259115219116, 1.4004826545715332, -1.0374863147735596, -0.8639960289001465, -0.34104520082473755, 0.9363328814506531, 1.0550516843795776, 0.44739076495170593, -0.5790168642997742, -1.1918598413467407, -0.3333972096443176, -0.9084360003471375, 2.225038766860962, -2.0118637084960938, 0.31363075971603394, 0.18257610499858856, 0.9955242276191711, 0.49992990493774414, -0.5085662007331848, 0.34252968430519104, -0.2918386459350586, -1.1086093187332153, -0.13632409274578094, 1.080804705619812, 0.4670930504798889, -2.400466203689575, 0.2440894991159439, 1.9895356893539429, -0.00983256846666336, 0.46753016114234924, 1.408835530281067, -0.19021688401699066, -0.694652795791626, 1.9210301637649536, 0.05292344465851784, 0.7541353702545166, 1.6017006635665894, -0.7742547392845154, -0.9111037850379944, -0.27547377347946167, -0.4186026453971863, -0.09035531431436539, 1.5528606176376343, -0.7087001204490662, 0.6582977175712585, 0.6506538391113281, -0.6810319423675537, 0.4253224730491638, 1.1355942487716675, -0.580371618270874, -0.2719976305961609, -0.0633755549788475, 0.35899436473846436, 0.3899565041065216, 0.30466151237487793, -0.36521223187446594, 0.45004817843437195, 2.502655506134033, 0.2509225308895111, -1.8040053844451904, 0.30221813917160034, -0.2775813639163971, -0.5136915445327759, -0.9627806544303894, 0.4036406874656677, 0.3833269774913788, -1.0045617818832397, 0.645793616771698, -0.5333463549613953, -2.588752031326294, -0.856563150882721, 0.04817330837249756, -1.139016032218933, 1.465793251991272, -0.6082304120063782, -1.2352232933044434, 1.0194495916366577, -0.2087688148021698, 0.04609159380197525, 0.610831081867218, 1.477582573890686, 0.4313259422779083, 1.899941086769104, -0.3098143935203552, -0.9538540244102478, 1.0114991664886475, -1.8453508615493774, -0.36577102541923523, 0.9452980756759644, 0.4434201121330261, 0.24323096871376038, 0.4901778995990753, 0.016698576509952545, -0.24083110690116882, 2.2589035034179688, -0.39777424931526184, -0.36757197976112366, -0.03869151324033737, -2.4972052574157715, 1.20423424243927, 0.5476049184799194, -1.5896930694580078, -0.11368090659379959, -0.5926849842071533, -0.1724807769060135, 0.4336203336715698, -0.7587308883666992, -1.0887796878814697, -0.6879271268844604, -1.157538652420044, 0.1774890124797821, 0.32791122794151306, 1.0521552562713623, -0.9152193069458008, 0.9702978134155273, -0.17635409533977509, -0.9625570774078369, 0.2997354567050934, -0.44059476256370544, -0.28724876046180725, 1.6046812534332275, 1.4837961196899414, -1.8470661640167236, -0.6053842306137085, 2.2887508869171143, 0.7726851105690002, -0.36989307403564453, 0.5544694066047668, -0.11626813560724258, 0.09833692759275436, 0.7378959655761719, -0.604246973991394, -0.18179014325141907, 0.8284587264060974, 0.07860103249549866, 0.07989799976348877, 0.17036756873130798, -1.232723593711853, 0.6127456426620483, 0.921629011631012, 1.5684776306152344, -2.7934916019439697, 0.03781989589333534, 0.7202577590942383, 2.100820302963257, -0.37544748187065125, 0.023678014054894447, -0.411419540643692, -1.4320111274719238, -1.9061566591262817, 0.5004321932792664, 1.3924847841262817, 1.2770625352859497, 0.10991016775369644, 0.4934455156326294, -0.5598801374435425, 0.7194462418556213, -1.9906772375106812, 1.7979775667190552, 0.21858923137187958, 0.12728658318519592, -0.6878976225852966, 0.3065328598022461, -0.4998549222946167, -0.5927635431289673, 1.478376030921936, 1.2876757383346558, 1.4304269552230835, 0.8979750871658325, 1.182221531867981, -0.3879644274711609, 0.08898237347602844, -1.3145633935928345, -0.20723223686218262, -1.3987658023834229, 0.09474316239356995, 0.2912661135196686, 1.0085976123809814, -0.36580049991607666, 0.0614393912255764, -0.016964923590421677, 0.012375196442008018, -0.8284064531326294, 1.5063611268997192, -0.9293450117111206, 0.3616741895675659, 0.4276314377784729, -0.7145614624023438, -2.733940839767456, 0.12761449813842773, -0.832309365272522, -0.2282135933637619, -0.20614565908908844, -0.6969445943832397, 1.371178150177002, -0.3117564618587494, 1.209007978439331, 1.2677823305130005, -0.5510547161102295, -0.5636333227157593, -0.025015726685523987, 1.817675232887268, 0.3535155653953552, -0.23011568188667297, -0.07548022270202637, -0.6292932033538818, 0.6934323906898499, 0.04033976420760155, -0.2365008443593979, -0.6017548441886902, -1.1776760816574097, -2.403710126876831, 0.5023213028907776, 1.172171950340271, 0.5184608101844788, -0.03586142510175705, -0.33994683623313904, -0.3241477310657501, 0.6842844486236572, -0.5365880727767944, 0.34397944808006287, 0.6096590757369995, -0.03076789155602455, 1.000726342201233, -2.5445170402526855, -0.31707438826560974, 1.8488070964813232, -0.9912949204444885, -0.8571867942810059, -0.4835593104362488, -0.9080910086631775, 0.7221129536628723, -0.9687429070472717, 0.8419979810714722, 0.9970343708992004, 1.1937012672424316, -1.2164289951324463, 0.2595767080783844, -2.3646469116210938, 0.565923273563385, -1.3498353958129883, 0.13519251346588135, 0.7353926301002502, 0.9693986773490906, 2.0855772495269775, 0.04582338407635689, 1.1652339696884155, -0.11275817453861237, 1.7207077741622925, -0.15290690958499908, 2.3199479579925537, 0.29798609018325806, -1.0121256113052368, -1.830365777015686, -0.11423850059509277, -0.6088449954986572, 0.36202308535575867, 2.07543683052063, 0.29719164967536926, -1.128594994544983, 1.0292335748672485, -0.6975803971290588, 1.2612274885177612, 0.040126509964466095, 0.16688111424446106, -1.1998374462127686, 0.5405244827270508, -0.5630120038986206, -0.8626359701156616, -0.9536477327346802, -0.8373897075653076, 0.5519446730613708, -0.028937440365552902, 0.9491790533065796, 1.1520675420761108, -2.0875611305236816, -0.8392558097839355, 0.37528136372566223, 1.4217116832733154, 0.006301391404122114, -0.4614296555519104, -0.6954250335693359, -0.6075822710990906, 0.2860671281814575, 0.15589319169521332, 1.1257753372192383, 1.7968981266021729, -2.032935380935669, 0.47835466265678406, -1.350772500038147, -0.49825969338417053, -1.2816885709762573, -0.9804419875144958, 1.1778234243392944, -0.6948063969612122, -0.5678523778915405, -2.3406929969787598, -1.087720513343811, -0.8866755366325378, -0.04956233128905296, -0.7794045805931091, -0.3980419337749481, 0.6400540471076965, -0.4480028450489044, 0.8033715486526489, -0.1572779268026352, 0.20609264075756073, 1.5889652967453003, 0.20226271450519562, -0.026677602902054787, 1.6647313833236694, -0.5001249313354492, 0.008430077694356441, 0.49838146567344666, -1.611955165863037, -0.09005539864301682, 0.3249276876449585, 0.5252441763877869, -0.00968609657138586, -0.6586852073669434, -1.0188316106796265, 1.3034025430679321, -0.7922360897064209, 1.0457618236541748, -0.791088342666626, -0.9236235022544861, 1.8943995237350464, 0.7299782633781433, 0.10859714448451996, -0.5619363188743591, 1.3621108531951904, -0.2760475277900696, -0.8721186518669128, -0.22183857858181, 0.832618236541748, 2.5176126956939697, 2.280010461807251, 1.4424077272415161, -0.8637387752532959, 0.439653605222702, -0.3421630859375, 0.21134966611862183, 0.8371352553367615, -1.2732139825820923, -0.12822578847408295, 0.3621222674846649, -0.8327049016952515, -0.24804261326789856, 1.2518999576568604, -0.3847540318965912, -1.0843631029129028, 0.050675250589847565, 0.7929211258888245, 0.9981973767280579, 1.3287391662597656, -0.3287622034549713, -1.1148319244384766, -0.1881307065486908, 0.14308011531829834, 1.2942235469818115, -0.20491616427898407, 0.5791425704956055, -1.300546646118164, 0.15178115665912628, 0.03740108758211136, -0.13688914477825165, 1.7823086977005005, 1.9246277809143066, -2.0158731937408447, 1.145580530166626, -0.8028604984283447, 2.807171583175659, 1.1708271503448486, 1.1232558488845825, 1.3025848865509033, 0.5867799520492554, -1.1808480024337769, -0.40465497970581055, -1.4588347673416138, 0.0932871401309967, -0.05232223495841026, 0.36332592368125916, 0.3561875820159912, -2.933393716812134, 0.24398840963840485, 2.1434876918792725, -0.9399756789207458, 0.2229299694299698, -0.22207523882389069, 0.18644686043262482, -0.0107297133654356, 1.7625893354415894, -1.5204421281814575, 0.7745163440704346, 0.2981133460998535, 0.0119944391772151, -1.151978850364685, 1.429201602935791, -0.6398494243621826, -0.040184251964092255, 0.8795099258422852, 0.2935521602630615, 0.6854822635650635, -0.34927260875701904, -0.4453865885734558, 1.0726100206375122, 0.004996239207684994, 0.8246157169342041, -1.5747114419937134, 0.7509483695030212, -0.4471506178379059, -0.20506814122200012, -0.8804599642753601, 0.38120734691619873, 0.8821251392364502, -0.682172954082489, -0.2961650788784027, 1.3593523502349854, -1.1848289966583252, 1.114068865776062, 0.22663602232933044, 0.8992182016372681, 2.1767566204071045, -0.9823724031448364, 0.10189523547887802, -3.2003962993621826, 1.9714548587799072, 0.8257585763931274, -0.47056373953819275, 0.3036699891090393, -1.044164776802063, -1.0008230209350586, -2.4461312294006348, -0.8834568858146667, -1.4484546184539795, 0.6409658193588257, -1.383570909500122, 0.21913382411003113, -0.7072171568870544, 0.5175043344497681, 0.08924872428178787, -1.1350528001785278, -0.3208673894405365, 0.009180519729852676, -0.49373799562454224, -0.16552874445915222, 1.537927508354187, -0.273832768201828, -0.5371788144111633, 0.03275905176997185, 1.0404843091964722, -0.0231403149664402, -1.256056785583496, 0.2627851068973541, -1.6608585119247437, 0.6794332265853882, 1.2491974830627441, -1.127432107925415, 1.6228148937225342, 1.947971224784851, -1.4320112466812134, 1.2766913175582886, -1.8071104288101196, -0.9784020185470581, 0.87636399269104, 0.2857741713523865, -0.7364705204963684, 0.4368766248226166, 1.4874993562698364, -0.886147677898407, -1.1703981161117554, 0.009095951914787292, -0.780665397644043, 0.30513373017311096, -0.6724725961685181, -0.6542671322822571, -1.0167924165725708, -0.8300963640213013, -0.18243353068828583, 0.4200313091278076, -0.5717144012451172, -0.3523221015930176, -0.6080871224403381, -0.46607357263565063, 1.051835060119629, 2.6121010780334473, 1.653331995010376, -1.4326046705245972, -1.6861546039581299, 0.7651346325874329, 0.9117594361305237, -0.07271434366703033, 0.8112785816192627, -0.45100584626197815, 0.5031245350837708, 0.20255832374095917, 0.17521947622299194, 0.8140844106674194, 0.07301414012908936, -1.3138272762298584, -1.83536958694458, 0.6513388156890869, 0.9850269556045532, 0.1359924077987671, -1.815834879875183, -0.8787015676498413, 0.4627837836742401, 0.5530007481575012, 1.206217646598816, 0.17577728629112244, 0.2923654615879059, -1.0781574249267578, -0.6587986946105957, 1.115382194519043, -1.1367672681808472, 1.0769814252853394, -0.13885007798671722, -1.343824028968811, 0.44746631383895874, -1.3706976175308228, -1.497144103050232, -0.10110457241535187, -1.3830344676971436, -0.37319883704185486, -0.7622343301773071, 0.6064795851707458, 0.13884924352169037, -0.24388322234153748, -0.3520483672618866, 0.3878980576992035, -0.10322262346744537, -1.043282151222229, 0.9857203364372253, 1.4377477169036865, -1.277345061302185, 1.115355134010315, -0.6481092572212219, 0.4619925320148468, -0.5371780395507812, 1.13953697681427, -2.2288124561309814, 0.016848284751176834, 1.0611944198608398, 1.5304088592529297, -0.07721863687038422, 0.9352626204490662, 0.25379112362861633, 0.8153294920921326, -0.8012394309043884, -1.8292156457901, 0.22299043834209442, -0.6774044632911682, -0.5343981981277466, 0.1931585669517517, 0.14705505967140198, 1.4071545600891113, 0.005629091523587704, 0.4241943955421448, -0.8529509902000427, -1.2902565002441406, 0.8406752347946167, 0.3998171091079712, -1.9571130275726318, -1.2255527973175049, 1.0109201669692993, -0.12058494985103607, -0.27078646421432495, 0.6115922331809998, -2.4424073696136475, 0.6142681241035461, -1.1535943746566772, 0.46589261293411255, 0.5858299732208252, -1.2603846788406372, 0.13735486567020416, 1.8942214250564575, 1.5755069255828857, 0.7205989360809326, -1.6481223106384277, -0.1316041797399521, 2.3341643810272217, -0.2020120769739151, 0.1590387523174286, -0.24622346460819244, -1.0125118494033813, -0.2615959644317627, 1.2234225273132324, 0.8728975653648376, -0.47122302651405334, -0.7649204134941101, -0.7410242557525635, -0.43116655945777893, 0.5509664416313171, 0.6538822054862976, -0.3673741817474365, 0.21354149281978607, -0.11798646301031113, 0.39537620544433594, -2.0815675258636475, -2.1505587100982666, -1.3544524908065796], [-0.4581265151500702, 0.9890773296356201, 1.263128638267517, 1.8053191900253296, 0.5114386081695557, 0.19478453695774078, -0.003124757669866085, 0.6719968318939209, 0.4368363320827484, 0.34071478247642517, -0.17337030172348022, -0.14217419922351837, -0.7056863903999329, -0.3092060983181, -0.002976011950522661, 0.09305330365896225, 1.1698356866836548, -0.08623231947422028, -0.7346782088279724, 0.06828594207763672, 2.143561363220215, -1.1115087270736694, -0.3604549467563629, 0.17423029243946075, -2.01928448677063, 1.2521122694015503, -0.2365604043006897, -1.9788700342178345, -1.6203315258026123, 1.3751707077026367, -1.2521438598632812, -0.017158623784780502, 0.3282966613769531, -0.479267418384552, -0.10412436723709106, 1.0287425518035889, 0.6262607574462891, 0.7030365467071533, 1.0331554412841797, 0.1693975180387497, -0.02354825846850872, -1.1767535209655762, -0.3537925183773041, 0.6056206822395325, 0.8413060903549194, -0.14476919174194336, 1.5887197256088257, -0.08357700705528259, -0.3925393521785736, -0.9883981943130493, 0.06134805083274841, 2.03127384185791, 0.7523989081382751, 0.5687264800071716, -0.9719167351722717, 2.232088088989258, -0.7989776134490967, -0.43978941440582275, -0.8707031607627869, 0.05938560888171196, -1.944432258605957, 0.3668474853038788, 1.1120662689208984, -0.27662327885627747, 1.6406030654907227, 0.02948562614619732, -0.9321509599685669, -0.954967200756073, -1.6261399984359741, 1.2030647993087769, -0.30418726801872253, 1.4347827434539795, -1.183586835861206, 0.886327862739563, 0.43606099486351013, 1.529030442237854, -0.09561661630868912, -0.6997956037521362, 1.4392353296279907, 1.0474773645401, -1.1422480344772339, -0.1909758746623993, 0.07521121203899384, 0.950874388217926, -1.316779375076294, -0.4663504660129547, -0.7896513342857361, -0.4532682001590729, 0.7856066226959229, -1.5570274591445923, -0.2195069044828415, -0.46612489223480225, 0.053380921483039856, 0.6589322686195374, 0.29787975549697876, 0.9679485559463501, 1.4878571033477783, -0.9574419856071472, -1.1061266660690308, 0.1619930863380432, -0.12179487943649292, -0.03829646110534668, 1.1637345552444458, -0.8313863277435303, 1.4126008749008179, 0.3060416281223297, 1.3548349142074585, -1.0709952116012573, -1.8672579526901245, 0.32863521575927734, 1.2212328910827637, -1.4614970684051514, 0.5013187527656555, -1.1286615133285522, 0.5629235506057739, 0.9319758415222168, 1.218646764755249, -0.44340986013412476, -1.8363511562347412, 0.517004668712616, 0.666239321231842, -1.4689393043518066, 1.5639243125915527, 0.4400031566619873, -1.0075410604476929, -0.06325770169496536, 0.7795637845993042, 0.6770980954170227, 1.0175622701644897, 1.04664146900177, -0.2269512116909027, -0.10074977576732635, -0.11999382823705673, 0.8378620147705078, -0.9726305603981018, 0.274297297000885, -0.03595627099275589, -2.180368423461914, 1.8063150644302368, 1.2440589666366577, -1.0607047080993652, -0.21459871530532837, 1.1819264888763428, -0.6025770902633667, -0.30164963006973267, -2.3876144886016846, 0.4470619559288025, -0.1480579972267151, 1.7455599308013916, 1.0143792629241943, -0.11367892473936081, 1.5377976894378662, 1.8099836111068726, 0.7268674969673157, -0.6102904081344604, 0.027501847594976425, -0.4318191409111023, 1.6603193283081055, 0.12348499894142151, -0.42435377836227417, 0.01706322468817234, -0.03541060909628868, -0.930704653263092, 1.1597752571105957, 0.6876716613769531, 1.0074957609176636, -1.4083094596862793, 0.6760196685791016, 0.6886093616485596, 0.5024541616439819, 0.36332330107688904, -0.2866438031196594, 1.7214851379394531, -1.7220227718353271, -0.49859318137168884, 1.3696259260177612, 0.22931602597236633, 1.0307239294052124, 0.25819626450538635, 1.01990807056427, 0.5989444255828857, 0.9048231244087219, -0.4911647140979767, -0.44814595580101013, -0.5227361917495728, -1.1779674291610718, 0.4191068112850189, -0.6336055994033813, 0.46674779057502747, 0.4985164999961853, 0.5165050029754639, 1.2491991519927979, 0.1781597137451172, -1.1829733848571777, -0.70902419090271, -2.210427761077881, -1.7552284002304077, -0.10676229745149612, -0.028694478794932365, -0.7757546901702881, -2.1411285400390625, 1.2939817905426025, -0.16044415533542633, -0.5645179748535156, 0.18942633271217346, 1.2894983291625977, -1.572975754737854, -2.142470359802246, 0.26678216457366943, 0.07361617684364319, -1.0393097400665283, 0.2064274698495865, -1.4274520874023438, -0.9713997840881348, -0.7773421406745911, -0.539416491985321, -1.146369457244873, -0.6306740045547485, -0.7924385666847229, 0.3811284005641937, 0.9143529534339905, 0.14046405255794525, 1.0515382289886475, -1.0336955785751343, 0.2280362993478775, 0.6970447897911072, 0.23374570906162262, 0.3886285722255707, -0.6544089913368225, -0.5209107398986816, 1.203700304031372, 0.11060603708028793, 0.9487237334251404, 0.3638797998428345, 0.8805333971977234, -0.11044277250766754, 0.18795756995677948, 0.07542077451944351, -0.12442632019519806, -0.6804504990577698, 0.5402255058288574, 0.9715527296066284, -0.28176435828208923, -1.133171796798706, 0.6784424185752869, 0.3293420374393463, 0.8244494199752808, 0.6171464323997498, 1.5250709056854248, -1.5631473064422607, -1.263657808303833, -0.38317587971687317, -0.3341681957244873, 0.27770277857780457, -0.5297878980636597, -1.1930687427520752, -0.07519357651472092, -1.4254542589187622, 0.07085026800632477, 1.3277082443237305, -0.6695930361747742, -1.0721927881240845, -0.6661154627799988, 0.905402660369873, -1.1321581602096558, -1.032412052154541, 0.34244224429130554, -0.4968734383583069, 0.1462322622537613, 0.11667071282863617, -0.35627663135528564, 0.0049369740299880505, -0.9384148120880127, 1.0061838626861572, 0.9516293406486511, 1.6369909048080444, 0.18774129450321198, -1.1882623434066772, 0.15703190863132477, 0.6136316061019897, 0.2608390152454376, 0.8217366933822632, -1.5299484729766846, -1.1390193700790405, 0.3667193353176117, 1.5193822383880615, 0.823502242565155, -0.43226492404937744, 0.16340109705924988, 1.5183852910995483, -0.3719453811645508, -1.085943579673767, -1.6799588203430176, -0.6642952561378479, 0.785147488117218, -0.7931534647941589, -0.10279015451669693, 1.4418704509735107, -0.06866806000471115, 0.05958089232444763, 0.15554775297641754, 2.23231840133667, -0.3341878652572632, -0.6251779198646545, -0.1923113763332367, -0.3435761034488678, -0.22044044733047485, -0.11644989997148514, -0.4071343243122101, -0.8967928886413574, -0.19186446070671082, -1.071921467781067, -0.23683016002178192, 0.7022680640220642, 0.3016182482242584, -2.109174966812134, 0.9190377593040466, 0.7267315983772278, 0.3329758644104004, -2.0965230464935303, 0.6550701856613159, -1.0913804769515991, 0.19002510607242584, -0.9248150587081909, -1.0744450092315674, 0.4810398817062378, 0.00512554869055748, -0.7815401554107666, -0.45218804478645325, 0.576553225517273, -1.1525216102600098, -0.7802791595458984, -1.7391009330749512, -0.5235161781311035, 0.6392619609832764, -1.1685922145843506, -1.4832775592803955, -1.7729135751724243, -1.7214016914367676, -1.1051260232925415, 0.7245615124702454, -0.47306352853775024, -0.40116026997566223, -0.7503834962844849, 0.7654773592948914, 1.0768036842346191, -0.34854891896247864, -1.176672339439392, -1.7692309617996216, -0.11345607042312622, -1.9573792219161987, -1.336807370185852, -0.13746120035648346, -1.0560623407363892, 2.8803467750549316, 1.4573887586593628, -0.10169930011034012, -0.842136561870575, 0.031508591026067734, 1.3173062801361084, 0.8998458981513977, -1.0278512239456177, 0.23238247632980347, -1.6788313388824463, -0.504778265953064, -0.4818507730960846, 1.3369721174240112, 0.5194262862205505, 0.45804840326309204, 0.9830554127693176, 0.18628981709480286, 0.6733380556106567, 1.1130337715148926, 0.25266581773757935, -1.0537489652633667, 0.5251135230064392, -0.5591458678245544, 1.076595664024353, 0.3085373640060425, -1.9145326614379883, -0.3956441283226013, -0.24116089940071106, -0.6205395460128784, -0.28707385063171387, -0.6905146837234497, 0.16532407701015472, 1.1492396593093872, -0.6262138485908508, -0.20731697976589203, -1.3643519878387451, -0.576890230178833, -2.0986757278442383, 1.3619601726531982, 0.9951019287109375, 0.5177651047706604, -1.6073259115219116, -2.2486116886138916, -2.2975165843963623, 0.7409324049949646, -0.7540563344955444, 0.43010208010673523, 0.3315305709838867, -0.5833667516708374, -2.776418685913086, -1.9774584770202637, -1.228959560394287, -0.6487293243408203, -0.9143442511558533, 0.8130753636360168, 0.8643076419830322, -0.5281768441200256, -1.8257066011428833, 0.011510717682540417, 2.0532381534576416, 1.3409881591796875, 0.9927365779876709, -0.9099129438400269, -0.574282169342041, -0.773343026638031, -0.40785112977027893, -1.3304489850997925, -0.8746684789657593, 0.18595369160175323, 0.885735809803009, -2.579514741897583, -0.5297321081161499, -0.06287430226802826, 1.2388174533843994, 0.5749852657318115, 1.6652312278747559, -0.3220977187156677, 0.7050814628601074, -0.5003586411476135, 0.186003640294075, -1.164004921913147, -0.4268963634967804, 0.8679714798927307, 0.48499295115470886, 0.4819808304309845, -1.3299187421798706, 0.45116081833839417, 0.33775410056114197, 0.08895687758922577, -0.0030416715890169144, 1.2526875734329224, -1.7592647075653076, -0.6155340075492859, 1.3684477806091309, 0.5821865797042847, 2.6339833736419678, 0.43958020210266113, 0.8622944951057434, -0.3578051030635834, -0.8926700353622437, 1.4324822425842285, 0.28128015995025635, 0.3633798658847809, -0.7885226011276245, -0.4164868891239166, -0.7858420014381409, -0.19100311398506165, 0.1821209192276001, 1.7047910690307617, -0.5695120692253113, -0.2973635792732239, -2.521790027618408, 0.805064857006073, 0.6788372993469238, -0.45513710379600525, 0.38635265827178955, -1.8339790105819702, -0.6329658031463623, 1.0157212018966675, -0.3201483190059662, -0.4528749883174896, 0.4183010160923004, -1.4681919813156128, 0.39071282744407654, -0.3274419605731964, -1.7759368419647217, -0.7020153999328613, 0.800283670425415, -1.9924490451812744, -0.9895212650299072, -0.5313205122947693, 0.059931714087724686, 1.1127487421035767, 1.904805302619934, -1.6553635597229004, 0.9160687327384949, 1.1303616762161255, -1.4735831022262573, 0.17992724478244781, -0.20232483744621277, -1.1868304014205933, 1.6857554912567139, 0.15704505145549774, 0.03749971091747284, -0.12521161139011383, -0.9550131559371948, 2.1598949432373047, -2.0423760414123535, -0.4868168830871582, 0.8419991731643677, 0.40976500511169434, -0.4088503420352936, -1.9680927991867065, -1.957348346710205, 1.2905193567276, -0.7868003845214844, -1.1654921770095825, 0.4299933910369873, -1.7874834537506104, -0.36042094230651855, -0.02508743293583393, -0.7613033652305603, 0.4677247107028961, 1.5917643308639526, 1.2246111631393433, 0.8298275470733643, -0.8128459453582764, -0.5513373613357544, -0.15509778261184692, -2.0837490558624268, 0.20885558426380157, -1.5176994800567627, 0.2609544098377228, -0.6684255003929138, 1.1497604846954346, 0.7308489680290222, 0.6858242750167847, -0.6731780171394348, 1.077628493309021, -0.9624570608139038, -0.5412032008171082, 0.012079307809472084, -0.9020477533340454, -1.1031402349472046, 1.6468313932418823, 0.19628813862800598, -0.16075122356414795, -0.4943751096725464, -0.8198119401931763, -0.9053923487663269, 0.13770225644111633, 1.4035162925720215, -0.48418575525283813, -0.16360105574131012, 1.5974323749542236, 0.20280376076698303, -0.4844065010547638, -1.0219697952270508, 1.675981044769287, -0.5770224928855896, 1.9681425094604492, -0.3409188985824585, 0.20918165147304535, 0.36083561182022095, 0.19847504794597626, 0.12668544054031372, 0.20127540826797485, -0.07625146955251694, 1.7354249954223633, -0.5344351530075073, -0.0005867388099431992, 0.4850911498069763, 0.591009259223938, -0.14820146560668945, 1.487661600112915, -0.691541314125061, 0.608226478099823, -0.6712267994880676, 2.0430049896240234, -0.24069736897945404, 0.48088082671165466, -0.7863513231277466, -0.47114163637161255, -0.3247552514076233, 1.6059296131134033, -0.13168096542358398, -0.23017407953739166, -0.4642687439918518, 1.1203229427337646, -1.4024758338928223, 1.0971599817276, -0.14821664988994598, 0.025124026462435722, 0.2233385592699051, 1.1614185571670532, 1.9451661109924316, 0.7592324614524841, -0.31070002913475037, 1.4105335474014282, -0.05086313560605049, -0.7926835417747498, 0.43577077984809875, 0.7693632245063782, 0.8776405453681946, -0.2392667531967163, -1.4533122777938843, 0.876982569694519, -0.2467915117740631, 0.41466978192329407, 0.3386309742927551, -0.6299360394477844, 1.5312868356704712, 0.3991987705230713, 3.1945104598999023, -0.097955621778965, -0.1657070815563202, 0.7786060571670532, -1.2401219606399536, 0.25703898072242737, -1.0742076635360718, -0.4565049111843109, -0.33090484142303467, -1.4189589023590088, 0.6722092628479004, 0.3480355143547058, 1.0723285675048828, -1.0830106735229492, -0.15270447731018066, 1.183124303817749, 0.6199270486831665, 0.06972701102495193, -0.3831470012664795, 0.640404224395752, -0.31422215700149536, 0.4713779389858246, 0.42161136865615845, -1.4035593271255493, -0.7209912538528442, 0.4055994153022766, 1.1398847103118896, -1.1377685070037842, 0.5484486222267151, -1.5148545503616333, -0.55069899559021, 0.7515236735343933, 0.04891359806060791, 1.4339408874511719, 1.0512824058532715, -0.15116845071315765, 0.20319482684135437, -0.32973533868789673, 0.028080955147743225, -1.6456935405731201, 0.8305953145027161, -0.833621084690094, -0.23837940394878387, 1.6807509660720825, 0.08274804055690765, -0.12782537937164307, 0.3922571539878845, 1.0278342962265015, 1.1653416156768799, 0.9860678315162659, 0.08947072178125381, -1.0819741487503052, 0.8408522009849548, 0.9074145555496216, -1.3500858545303345, -2.022359848022461, -0.5936416983604431, -1.8128176927566528, -0.652919352054596, 0.8716589212417603, 0.19162631034851074, 0.2555329501628876, -0.9449860453605652, 0.6964095830917358, 0.45354241132736206, 0.5681830048561096, -0.9805417060852051, -0.8629928231239319, 0.0826411247253418, 0.6668784618377686, 0.7309631109237671, -0.4935513436794281, 1.2918791770935059, 1.0344330072402954, 2.4245216846466064, 0.5172359347343445, -1.379744052886963, -0.635899543762207, -1.2215873003005981, 0.8381614685058594, -1.2625510692596436, -1.5739444494247437, 1.8849730491638184, 0.25458410382270813, 0.9451988935470581, 0.7612435817718506, 0.2982657551765442, 0.9326956868171692, -1.499794602394104, -0.4093429446220398, 0.3329896032810211, 1.5070114135742188, -0.01701098494231701, -0.49168679118156433, 0.2166816145181656, 1.374130129814148, -0.14187300205230713, 0.1069364920258522, 0.3311532139778137, 0.7413129210472107, 0.5387729406356812, -1.3231407403945923, 0.5112639665603638, 0.29859602451324463, -0.10777628421783447, 1.2794575691223145, -0.5169651508331299, 1.2391135692596436, 0.5857949256896973, -0.32518070936203003, 1.224545955657959, -0.058305997401475906, -0.400813490152359, -0.4538091719150543, -0.37290874123573303, 0.5798861384391785, -1.7811636924743652, 1.0236437320709229, -0.2653302252292633, -0.7113785147666931, -0.1540570706129074, 0.17525646090507507, 0.09148025512695312, 0.694156289100647, -1.0358679294586182, 0.8922188878059387, 0.15160110592842102, -1.6804587841033936, 0.7095122933387756, -2.5831804275512695, -1.0332635641098022, 1.547445297241211, 0.2964438498020172, -0.9831556081771851, 0.32382991909980774, 1.4144787788391113, -0.7332866191864014, -1.287553071975708, 1.0583595037460327, -0.9338532090187073, 0.40277114510536194, -1.4645277261734009, -0.48883742094039917, 0.11517096310853958, -0.6684885025024414, -0.09048325568437576, -0.5038347244262695, -1.0772535800933838, 0.25158944725990295, 0.3975497782230377, -0.6726413369178772, -0.6922227144241333, 1.4247828722000122, 0.17049440741539001, -0.867241382598877, 1.5436004400253296, 0.08799055963754654, 0.5146345496177673, 0.9194742441177368, 0.41453006863594055, 1.078994631767273, 0.32773125171661377, -0.8269503116607666, 0.34322530031204224, -1.1451294422149658, 0.2540656328201294, -1.9074679613113403, 0.39033615589141846, 0.02301156520843506, 0.749893307685852, 0.20840388536453247, -0.41157883405685425, -0.8075819611549377, -1.3810679912567139, -0.5577498078346252, 0.736701488494873, -1.7402251958847046, -0.031104909256100655, -1.212748646736145, -0.6373834609985352, 2.2348885536193848, 0.04206792637705803, 0.8328230977058411, 2.0649807453155518, -0.33514246344566345, -0.17654357850551605, 0.7661810517311096, 0.25941014289855957, -0.14916059374809265, 0.11169398576021194, 0.41235044598579407, -0.39705878496170044, -0.6620428562164307, -0.7819617390632629, -1.305755615234375, -0.20720306038856506, -0.1529282033443451, 0.9114770889282227, 0.8452580571174622, 1.6294896602630615, -0.05509818717837334, -1.1457968950271606, -1.4976774454116821, 0.2370780110359192, 0.13605134189128876, 1.1379368305206299, 0.47633200883865356, 1.3322603702545166, -0.2611365020275116, -0.8814375400543213, 0.3719824254512787, -0.2884003221988678, -1.6202079057693481, 0.0193785447627306, -1.5899198055267334, 0.4977751672267914, 0.21031902730464935, 0.381719172000885, 0.6898313760757446, -1.6577881574630737, -0.2944534420967102, 0.058143727481365204, -0.5627697110176086, 0.547027587890625, -0.4816792607307434, 2.159956455230713, -0.11377803981304169, -0.6762022972106934, 0.7157490253448486, 1.4224101305007935, -0.06242981553077698, -0.15432503819465637, 0.38700786232948303, 1.1870415210723877, -0.8471660614013672, -1.5671013593673706, 1.3509753942489624, -0.3768705427646637, -1.236443281173706, -0.13257087767124176, -1.100549578666687, 0.9323663115501404, -0.7623332738876343, -0.02201438695192337, 0.6703042984008789, 0.01484882365912199, -0.7390769720077515, 0.3642314672470093, 0.531639814376831, -0.39372941851615906, -0.2484467774629593, -1.1480202674865723, -0.7629261612892151, 1.5984256267547607, 0.4366213381290436, -0.18997584283351898, 1.3058041334152222, 0.34323766827583313, -0.3785097897052765, -0.8110072016716003, 1.0283896923065186, -1.4630415439605713, -1.4568161964416504, -0.05949847027659416, 1.430662751197815, -0.7608531713485718, -0.35600775480270386, -1.4701628684997559, 0.219110369682312, -1.0084795951843262, -0.2443845272064209, -0.5671201348304749, 0.5598440766334534, 0.7008417248725891, -0.3993242681026459, -2.049450635910034, -0.8269044160842896, -0.888359546661377, -0.34816744923591614, -0.6350658535957336, 0.6219491362571716, -1.8951835632324219, 0.17165933549404144, 0.20905160903930664, -0.06565523147583008, -0.36018484830856323, 0.5788761973381042, 0.2938995659351349, 0.5972164273262024, -1.4737073183059692, 0.8728974461555481, 0.7154845595359802, -1.550135850906372, -1.010564923286438, 0.6720244288444519, 0.7608916759490967, 0.3824111819267273, 0.430208683013916, 0.9802479147911072, -0.04612239450216293, -0.16661900281906128, 0.4796404242515564, 2.319282293319702, -1.8922300338745117, -1.142928123474121, -1.1123309135437012, -0.9751582741737366, -1.257502794265747, -0.9320213794708252, -0.3928685486316681, -0.10302122682332993, 0.12142225354909897, -0.804771363735199, -0.23956623673439026, 0.5752953886985779, -0.06554652005434036, -0.04588764160871506, -0.6562183499336243, 0.7356764674186707, -0.24674323201179504, -1.4442319869995117, -0.3543814420700073, 1.0406299829483032, -1.8794827461242676, -0.16133131086826324, 1.1280866861343384, 1.1002615690231323, -0.4420578181743622, 0.07926435768604279, -1.6014611721038818, 2.511441946029663, -1.424338459968567, 0.732694149017334, -0.4098846912384033, 0.27774620056152344, 0.7161870002746582, 1.798914909362793, -1.218399167060852, 0.33163896203041077, -0.1085604727268219, 1.1987977027893066, -0.349341481924057, 0.6151569485664368, -0.8524725437164307, 0.37500691413879395, -0.027361975982785225, 0.6626094579696655, 0.9749605655670166, 0.1973300725221634, -0.5660046935081482, -1.8071283102035522, 0.47705841064453125, -0.5221623182296753, -1.1432079076766968, 0.21765896677970886, -1.3943977355957031, 1.5887404680252075, -0.3985419273376465, 1.7593389749526978, -1.5047379732131958, 0.2060409039258957, -0.5877235531806946, -0.8308402299880981, -0.3574383556842804, 0.17619743943214417, 1.8584277629852295, -1.7961050271987915, 0.4602799713611603, 1.769260287284851, -0.5678619146347046, -0.8806290030479431, -1.536538004875183, -1.2168694734573364, -0.47001683712005615, -2.217522144317627, 0.30446088314056396, -0.32804247736930847, -1.3952735662460327, -1.2591843605041504, 0.03459109365940094, -0.9242740273475647, -0.4942733645439148, -0.4785069227218628, 1.499809980392456, -1.4365777969360352, -0.16550494730472565, -1.585273265838623, 1.1947972774505615, -0.3323825001716614, 0.5142672061920166, 2.009793281555176, 0.5333996415138245, -0.5357044339179993], [0.540295422077179, 0.2905067205429077, 1.1669480800628662, -1.3853065967559814, 0.815666675567627, 0.1745261251926422, 0.21987110376358032, 1.0963633060455322, -0.38302284479141235, -0.6227248311042786, 0.6255455613136292, -0.28140881657600403, 1.0467040538787842, 1.2389343976974487, 1.0887471437454224, 1.4505118131637573, 0.5823067426681519, -1.9095983505249023, -0.14502164721488953, 0.6860028505325317, 0.3634188771247864, 1.240045428276062, -3.2352957725524902, -1.3911173343658447, -1.4195773601531982, -2.139925956726074, -0.05428863689303398, 0.6527719497680664, -0.2152937799692154, -0.012542052194476128, 1.0767680406570435, -0.1576276570558548, -3.06306791305542, -1.441424012184143, -0.5810657143592834, -0.35581138730049133, 1.733939290046692, -0.3415195047855377, 0.550432562828064, -1.2200958728790283, -0.9456236362457275, 0.1746399998664856, 0.5592071413993835, 0.376869261264801, 0.1054750382900238, 0.7401971817016602, 0.7126677632331848, 0.05783643573522568, -0.780055582523346, -0.11872392892837524, -1.929829478263855, -0.4750438332557678, -1.300665259361267, 0.3272165358066559, 0.6245493292808533, 1.9510141611099243, 0.728093147277832, 0.47289615869522095, -0.23778444528579712, -1.172606110572815, 1.4972472190856934, -1.222379446029663, -0.4590564966201782, -2.032099485397339, -0.2663210928440094, -0.6572374701499939, 0.340313196182251, -0.2506421208381653, 0.6698288321495056, -1.8005815744400024, 0.05743369087576866, -0.19177491962909698, -0.6295262575149536, 0.2547837197780609, 1.2760487794876099, -0.057327114045619965, -1.997586727142334, 0.05893925204873085, 0.20538519322872162, -1.052805781364441, -0.8453502655029297, -0.5431012511253357, -1.1605490446090698, -0.15782158076763153, 0.826287031173706, -0.9542961716651917, 0.045400962233543396, 1.4187841415405273, -0.7606323957443237, -0.47939378023147583, -0.35687941312789917, -0.20082491636276245, -0.9411444664001465, 0.12256991118192673, 2.022982358932495, 0.4965047240257263, 0.07296527177095413, 1.6378545761108398, -0.6404243111610413, -0.653252124786377, -0.08084400743246078, -0.8769210577011108, -1.6125237941741943, -0.43349212408065796, 0.39893510937690735, 0.7202115654945374, -0.28527799248695374, -0.3644544780254364, 1.0723143815994263, -1.494762659072876, 0.48703649640083313, -0.05120278522372246, 1.1101182699203491, 0.3143959641456604, -0.01245631743222475, 0.9658513069152832, -0.1396019011735916, -0.41398540139198303, 1.7459067106246948, 1.1458793878555298, -1.402449607849121, -0.1740693598985672, -0.9112259149551392, 0.005806545726954937, 0.010882243514060974, -2.012542724609375, -0.9752019643783569, -0.6632328629493713, -0.8443689942359924, -1.4421194791793823, 1.3142962455749512, 0.1462131291627884, -1.9674304723739624, -0.015853019431233406, 1.5282050371170044, -0.15042361617088318, 1.9255940914154053, -1.0611318349838257, -0.06211460009217262, 1.7518383264541626, 0.45860734581947327, 0.4867825508117676, 0.08627393841743469, 0.8553915619850159, -1.0088075399398804, 1.7298870086669922, -0.6501445770263672, -0.2280389815568924, 1.9470341205596924, -0.1789868175983429, 0.2559705972671509, -0.4002108573913574, 1.6855136156082153, -0.49545368552207947, -0.17447108030319214, 0.19934715330600739, -1.4920581579208374, 0.8359156250953674, -0.1068030297756195, 0.17746101319789886, 0.7865434288978577, -0.9333013296127319, -0.34157028794288635, 0.29067492485046387, 0.41416484117507935, -0.7803098559379578, 0.5395651459693909, 0.027210716158151627, -1.5032674074172974, -0.7361867427825928, 0.23650936782360077, -0.9320375919342041, -0.67099928855896, -1.2348374128341675, 0.35830116271972656, 0.006816724315285683, 0.3376893103122711, 0.19883708655834198, -0.9700287580490112, 0.021407322958111763, -0.46221157908439636, -0.4883885383605957, -0.028146883472800255, 0.7503147125244141, -0.6193320751190186, 0.05521273612976074, -1.4462993144989014, 0.7784655690193176, 0.6705209612846375, -0.8254567384719849, -0.7352533936500549, 1.583681344985962, 0.7010731101036072, -1.0045466423034668, 1.7985986471176147, 1.1275254487991333, -0.45863696932792664, 0.15650054812431335, -0.4717479348182678, -0.48691296577453613, 0.09963352978229523, 0.08382336050271988, -0.11085999757051468, -0.7307067513465881, 1.310142159461975, 0.01795082725584507, 0.8154043555259705, -0.9005814790725708, -2.033750534057617, -0.3539932370185852, 0.33509084582328796, 0.4974982440471649, 1.6081088781356812, -0.19158972799777985, 1.2194550037384033, 0.6412718296051025, -0.8243376016616821, 0.6644511818885803, -0.7728644013404846, 0.5500138401985168, 1.2124074697494507, 0.6919521689414978, -0.3968871831893921, 0.3981838524341583, 0.9158183932304382, 0.3032917380332947, -0.9848142862319946, -0.1623939871788025, 0.042170729488134384, -0.17276179790496826, 0.11731065809726715, 1.389442801475525, -0.09041188657283783, 0.16716523468494415, -0.4638002812862396, -0.3951479196548462, 0.5485577583312988, -2.01611065864563, -1.2026586532592773, -1.9419598579406738, 0.6697362065315247, -0.3241327106952667, -0.16397413611412048, -0.5962501764297485, 1.4161893129348755, -0.41491612792015076, -0.5111032128334045, -1.238515019416809, 1.913965106010437, -0.507638692855835, 0.3465100824832916, -0.8830458521842957, 2.3388965129852295, 1.9702943563461304, 0.80253005027771, -1.0123357772827148, -1.526055097579956, 0.7361648082733154, 0.5670537948608398, 0.06648880243301392, 0.19494463503360748, 1.0426338911056519, -0.6225130558013916, -0.8530195951461792, -0.6985772848129272, -0.759418249130249, 0.18986253440380096, -0.35229936242103577, 0.025530336424708366, 0.5874320268630981, 0.7000241279602051, -0.1086122915148735, -1.7279672622680664, -0.609799861907959, 0.916275680065155, -0.509589672088623, 0.5529203414916992, 0.7399219274520874, 0.9807668924331665, -0.35604622960090637, -0.9787412881851196, -0.3550235629081726, 0.07699936628341675, 0.3076145052909851, 0.6038210391998291, 0.9429972767829895, -0.09833390265703201, -0.8562301993370056, -1.4778680801391602, -0.9816736578941345, -1.950932502746582, 1.2366657257080078, 0.5128071904182434, 0.36129993200302124, -0.43353933095932007, 1.4512144327163696, -1.6573221683502197, 0.0818210020661354, -0.6358964443206787, -0.9110068082809448, 0.2598053812980652, 1.2369117736816406, 0.33591732382774353, 0.11981328576803207, -0.0521627776324749, -0.7019801139831543, -0.3577409088611603, 0.5719793438911438, -1.3920868635177612, 1.2878319025039673, 0.6362565159797668, -0.4702305495738983, -1.3719887733459473, 0.6048645377159119, 0.17423175275325775, -0.5680094957351685, 0.36394238471984863, -0.2996950149536133, -0.12298554182052612, -0.7661082148551941, -0.2533321976661682, 1.1546937227249146, 0.34885454177856445, -0.11727762967348099, -1.562712550163269, -0.780441403388977, -0.9264428019523621, -0.3298361599445343, 2.325035572052002, 1.0884052515029907, 0.6259685754776001, -0.5347570180892944, -0.43175289034843445, -0.20602601766586304, 0.2019253671169281, 1.8039662837982178, 0.7328510284423828, 1.4645074605941772, -0.04784355312585831, -0.15291409194469452, 0.8069902658462524, 1.0952857732772827, -0.45817863941192627, -0.6899901628494263, -1.5703381299972534, -0.9570823311805725, -1.1610080003738403, -1.3215117454528809, -0.24537791311740875, -1.6207239627838135, -0.2757223844528198, 1.5463831424713135, 0.14299803972244263, -0.9807489514350891, 2.510817766189575, 1.5420359373092651, -0.40814024209976196, 0.4036911725997925, -0.8497993350028992, -0.2723104655742645, -0.4591400623321533, -0.3118099868297577, 0.1737561821937561, 1.4065405130386353, 0.3741707503795624, -0.3492872416973114, -0.6226120591163635, -0.5657581686973572, -0.4644964635372162, -2.195603609085083, 0.023283280432224274, -0.607767641544342, 0.01570091024041176, 0.4953439235687256, 0.5694862008094788, -1.6522796154022217, -0.4327336847782135, 0.7628075480461121, 0.21379394829273224, -1.2323487997055054, -0.6519952416419983, -0.4285765588283539, 0.017705556005239487, -1.3427150249481201, -1.9086837768554688, -1.157034993171692, -0.6012499928474426, -2.7683277130126953, 1.5426311492919922, 0.4048430919647217, -0.8789618611335754, 0.8383277058601379, -0.367068350315094, 0.40352103114128113, 1.822609305381775, 1.1327210664749146, 0.28834667801856995, -1.2525098323822021, 1.5603036880493164, 0.6559662222862244, -1.1148074865341187, 1.5960237979888916, 1.0087474584579468, -1.8577377796173096, 0.6853972673416138, -0.2650064527988434, 0.45293527841567993, 1.3760241270065308, -0.49392038583755493, -0.7285597920417786, 1.6823432445526123, 2.06817626953125, -0.01409489382058382, -0.9170590043067932, -0.9965346455574036, -0.5910828709602356, -2.2145891189575195, 1.8856844902038574, 0.3252107501029968, 0.7899391651153564, 0.11215253919363022, -1.3425549268722534, 0.782214343547821, 0.9375211596488953, 1.077661156654358, -0.44805195927619934, -1.130868911743164, 0.14232154190540314, -0.39430680871009827, 0.684908390045166, -0.04280626028776169, -0.6822320818901062, 0.5632477402687073, -0.22953033447265625, 0.9745298624038696, -0.18054722249507904, 2.3394463062286377, 1.5892075300216675, -0.34204962849617004, -0.8318293690681458, 0.06254608929157257, 1.0956554412841797, 0.14398998022079468, 0.03882112354040146, 0.8728649020195007, 0.8006142377853394, -0.44062677025794983, 0.5888646841049194, 1.2174652814865112, 1.2060421705245972, 1.275100588798523, 0.38981255888938904, -0.05546313524246216, -0.622830867767334, 0.22350464761257172, 1.012550711631775, -1.1578783988952637, -1.0190001726150513, 1.1660010814666748, 0.628716766834259, 1.3563368320465088, 0.45996782183647156, 1.1327366828918457, -0.17793728411197662, 2.067131280899048, 0.2721908688545227, 0.04726416617631912, -0.1266050487756729, -0.8312755227088928, -0.10650526732206345, -0.9082639813423157, 0.13184380531311035, 0.27358338236808777, 0.07450102269649506, 0.40687909722328186, -0.05505175143480301, 0.07925690710544586, 0.01348186656832695, -1.4097154140472412, 0.5522504448890686, -2.261171340942383, -1.2840253114700317, -0.10116714984178543, -0.07716087996959686, 0.2247689664363861, -0.9683232307434082, 0.19714295864105225, -1.9555108547210693, 0.34263908863067627, -0.9716455936431885, 0.4595699608325958, -0.5189497470855713, -0.012300863862037659, -0.3442321717739105, 0.9406741261482239, -0.3027680814266205, -0.9134733080863953, 0.41930723190307617, -1.148481011390686, -0.608320415019989, 0.7086571455001831, -0.059859927743673325, -0.9459572434425354, 2.2164411544799805, 0.7868728637695312, 1.17341148853302, 0.6784351468086243, 0.5133897066116333, -1.4556339979171753, -0.16506938636302948, 0.047211844474077225, 0.7809041142463684, 0.8268107771873474, -0.8106734752655029, -1.9999114274978638, -0.5109542608261108, -0.13148781657218933, 0.23477119207382202, -0.018720975145697594, 2.461102247238159, -0.39086732268333435, 0.0747874304652214, 0.9793640375137329, -1.1540125608444214, -0.5804724097251892, 1.0877604484558105, -1.2439751625061035, 0.8262211084365845, 0.8665529489517212, -0.721195638179779, -0.8497428894042969, 1.276336431503296, -0.126628577709198, -1.6751042604446411, 0.09730324894189835, -0.3350488841533661, 0.4565804600715637, -0.8317450284957886, -0.7535540461540222, -1.2014847993850708, -2.372659206390381, 0.5283048152923584, 1.7583526372909546, 1.0238680839538574, 0.3350415825843811, -0.8244243860244751, 0.48885634541511536, -0.6173450946807861, -2.3045029640197754, 0.5969284772872925, 0.10152072459459305, -0.9758756756782532, 0.612354040145874, -0.3669586777687073, -0.0023547157179564238, 0.2645414471626282, 0.30166831612586975, 0.1835978776216507, 0.692382276058197, 0.8425902128219604, -0.04544684290885925, 2.418700933456421, -0.3858301043510437, -0.48892250657081604, 1.1434928178787231, -1.8972768783569336, -1.5473893880844116, 0.16530606150627136, -2.1003355979919434, -0.22523726522922516, -0.7236049771308899, 0.865172803401947, 1.4717742204666138, -0.40545177459716797, -0.9730819463729858, -0.6364074945449829, 0.6091274619102478, -1.1846050024032593, -0.952191948890686, 0.7995598912239075, -0.34662118554115295, 0.012026848271489143, 0.23950548470020294, -0.7318230271339417, 0.8239198923110962, 1.7377017736434937, 1.2438511848449707, 1.5189529657363892, 0.7457316517829895, -1.487790822982788, -1.2872695922851562, -0.9434548616409302, -0.03175466135144234, -0.14222882688045502, 0.3056359589099884, -0.6203898191452026, -0.4865320026874542, -0.2437717616558075, -1.5174137353897095, -1.3270156383514404, 0.2980602979660034, -1.1320055723190308, -1.9073562622070312, -0.22805356979370117, 1.1515928506851196, -0.4692403972148895, 0.1685352623462677, -1.3035593032836914, 0.31508582830429077, 1.4997355937957764, -1.4996821880340576, 1.4695241451263428, -1.1744095087051392, 0.6354829668998718, -0.784570574760437, -0.11730015277862549, -0.07394562661647797, 1.0615522861480713, 0.06806321442127228, -0.5648859739303589, 1.3607274293899536, -0.6492194533348083, -0.2038220763206482, 1.6309903860092163, 0.6126646995544434, -1.2569663524627686, 0.8656641840934753, 1.862321376800537, -0.5892370939254761, -0.0015942594036459923, -0.8124740123748779, 0.42872390151023865, -0.2213030606508255, 1.0586965084075928, -0.3564428985118866, 0.4623860716819763, -0.2539152503013611, -2.176910877227783, 1.8780890703201294, -0.6401638984680176, -0.11628226935863495, -0.9274123907089233, -0.550954282283783, 0.602227509021759, -0.6770206093788147, 0.5332649350166321, 1.1578322649002075, 0.17774026095867157, 0.26734668016433716, -0.31151455640792847, 1.627176284790039, 1.2651299238204956, -0.43198704719543457, -0.7271518111228943, 0.9695253372192383, -0.8907552361488342, -2.1878254413604736, -0.6188251972198486, 0.3060433864593506, 1.006145715713501, -0.08316131681203842, 2.04835844039917, 1.8226627111434937, -1.0691635608673096, 0.4931609630584717, 0.4625925123691559, -2.4186043739318848, 1.8368842601776123, -1.6059941053390503, -1.389984369277954, 0.9454094767570496, -0.4978500306606293, -0.654237687587738, -0.7649188041687012, -1.3602266311645508, -0.13551044464111328, 0.2979637682437897, 0.14955686032772064, -1.8211065530776978, -0.1601932793855667, -0.6882977485656738, 0.41443300247192383, 3.030529499053955, -0.8966742753982544, 0.3610382378101349, 0.4925515949726105, 0.5860958099365234, -0.712279736995697, -0.3366057872772217, -1.6067018508911133, 0.22203969955444336, -0.2663184106349945, -1.021590232849121, -0.1792624294757843, -0.08648204058408737, -1.6625592708587646, -0.02230912260711193, 1.8564338684082031, -0.25290751457214355, -1.4712109565734863, 1.1481717824935913, -0.8637229204177856, 0.2975420653820038, 0.1975325047969818, 0.6669600009918213, -0.12629400193691254, -0.6802175641059875, -1.0801606178283691, 0.9837974905967712, 1.3047035932540894, 0.06531696766614914, -0.18355326354503632, -0.4173811078071594, 0.318651020526886, -1.906782865524292, -1.32389235496521, 0.7187536954879761, 0.2563587725162506, 0.49203774333000183, 0.08454541862010956, 0.5623704195022583, 1.2182683944702148, -0.8441691994667053, 1.3207390308380127, -0.5172097086906433, -0.20588187873363495, 0.7627542018890381, 0.052176568657159805, 0.4976648688316345, 0.041280537843704224, -0.2163313329219818, -0.3754621148109436, 0.2914993464946747, 0.5259740948677063, 0.046603068709373474, -0.843246340751648, 0.05612105131149292, 1.516772747039795, -0.3845787048339844, -0.1096317395567894, -0.2736665904521942, -0.13327397406101227, -1.330902338027954, 1.6137033700942993, -0.2500523328781128, -0.4163941442966461, -1.1917489767074585, 0.8389222025871277, 0.481668084859848, -0.5282162427902222, -1.118457317352295, 3.821982681984082e-05, 0.8848061561584473, -0.33802953362464905, -1.412379264831543, 0.9038344621658325, -0.5349957942962646, 0.14338372647762299, -0.056205686181783676, 2.3488166332244873, 0.1293872594833374, 0.5308789610862732, -1.0467921495437622, 0.8923043012619019, 2.196232795715332, 1.0736075639724731, 1.567445993423462, -0.1609024703502655, -0.5300077795982361, 0.7499266862869263, 0.25093671679496765, -0.45015954971313477, -1.2065025568008423, -1.7875558137893677, 0.09435915946960449, -1.2077052593231201, 0.6541674137115479, -0.3164784014225006, -0.3725120723247528, -1.2827705144882202, -1.1555802822113037, 0.7382892966270447, 0.3282836973667145, -1.6157195568084717, 0.7847578525543213, -1.173935055732727, 0.44708913564682007, 1.306564450263977, 0.35048848390579224, 0.27274996042251587, -0.39379778504371643, -0.4299943149089813, 0.16951589286327362, 0.05432553589344025, 0.833962619304657, 0.02242632396519184, -0.42818087339401245, 0.8709550499916077, 0.11709728837013245, -1.0237776041030884, -0.8591864109039307, -0.12604627013206482, -0.08453398197889328, -0.04584617540240288, -1.1839646100997925, 2.103783130645752, 1.2058334350585938, -0.7865393161773682, 0.619211733341217, -0.6342817544937134, -0.7355892658233643, -1.1061025857925415, 0.27778440713882446, -0.42873698472976685, 1.4330494403839111, 0.547119677066803, -0.4058171510696411, 0.5804488062858582, -0.7479453682899475, -0.20884814858436584, -0.1224851980805397, -0.2148623764514923, -0.7627580761909485, -0.33118242025375366, 0.3702735900878906, 0.6162349581718445, 0.9290403723716736, 0.008019126951694489, -1.5772483348846436, -0.643705427646637, -0.4275427460670471, -0.6634271144866943, -0.5783008337020874, -0.5529397130012512, 0.03225329518318176, 0.3210282325744629, -0.2538410723209381, 0.9592092037200928, 0.3090718388557434, -0.22943738102912903, -0.6285645961761475, -0.19591745734214783, -0.9453482031822205, 1.003747582435608, 0.3211454153060913, -0.04237405210733414, 1.4242663383483887, 0.8890492916107178, 0.7401383519172668, -0.12038378417491913, -1.0481544733047485, 0.4505520462989807, -0.1005365252494812, -1.030493974685669, 1.3393713235855103, -1.4772632122039795, 0.6705996990203857, 1.978107213973999, 0.17373692989349365, 0.23324808478355408, -0.49126747250556946, -1.6432137489318848, -0.10288713872432709, -1.0083613395690918, 0.2006516307592392, 2.468928575515747, 1.3384736776351929, 1.69309663772583, 0.9624184370040894, -0.6118248105049133, 0.42011719942092896, 1.383204460144043, -0.9268582463264465, 0.3923870623111725, -1.5959492921829224, 0.9840339422225952, -0.12215038388967514, 0.8053417801856995, 1.3888232707977295, -1.7875926494598389, 0.018078405410051346, -0.40796786546707153, 0.4305386245250702, 0.27023184299468994, 0.4946576952934265, -0.35571831464767456, -1.2372517585754395, 0.25139302015304565, -0.9348138570785522, -0.2660026252269745, -1.4156551361083984, -1.6375865936279297, -0.48911046981811523, 0.7613219022750854, 1.1521238088607788, -0.11365219950675964, -0.2848508358001709, 0.7022144198417664, 0.15586698055267334, -2.30757999420166, 1.4851868152618408, -2.2060301303863525, -0.47861939668655396, 0.9181740283966064, 0.225407674908638, -1.8770655393600464, -0.9951276779174805, -0.39009231328964233, -0.6393885612487793, 1.3255698680877686, -1.2749836444854736, -1.0363121032714844, -0.7288041710853577, 1.3237671852111816, -0.4811016023159027, 0.5255954265594482, -0.4744720160961151, -2.1396665573120117, 0.10916158556938171, -0.8994854688644409, 0.00016430031973868608, -0.6908375024795532, -0.6926791667938232, 0.8121722340583801, 1.4524821043014526, 0.9567784667015076, -0.1271294802427292, 0.32669875025749207, 1.4221211671829224, -0.522605836391449, -1.9626284837722778, -1.2590925693511963, -0.8587946891784668, 0.9443383812904358, 1.9744905233383179, -1.1169543266296387, 0.37588566541671753, -0.2640867531299591, 0.5043148994445801, -1.3467175960540771, -0.18605241179466248, 1.3892465829849243, 1.9255999326705933, 0.045870278030633926, 0.7966734766960144, 0.8687462210655212, -1.0003794431686401, -1.05753755569458, 0.3102535903453827, -1.2149772644042969, 1.3309448957443237, 3.3690781593322754, -1.1878808736801147, -0.4395410120487213, -1.903287410736084, -1.6810853481292725, 0.47207415103912354, -0.5542517304420471, 0.2736707925796509, -0.7229664325714111, -1.0619956254959106, -1.442030906677246, 3.1503467559814453, -0.5224152207374573, -0.34457382559776306, -1.5734394788742065, 0.377432644367218, -2.1328766345977783, 0.9504634141921997, -0.2925224006175995, 1.3446813821792603, -2.1592507362365723, 0.25317618250846863, 0.07545462250709534, -2.3956425189971924, 0.0832698866724968, -0.8257519602775574, 0.9265581369400024, 1.597799301147461, 0.8571738004684448, -0.727668285369873, 1.2329789400100708, -0.17117907106876373, 0.2029346525669098, 0.8468832969665527, -0.21577872335910797, -2.146144390106201, 1.9037688970565796, 0.597395122051239, 0.9843863248825073, -0.22165821492671967, 0.5435442328453064, -0.5522396564483643, -0.4947073459625244, 0.26279735565185547, 0.7916186451911926, -0.5779176354408264, 0.9972298741340637, -2.1933979988098145, -1.0572232007980347], [-1.15272855758667, 1.9931817054748535, 0.6353570222854614, -0.37192821502685547, -0.6787083745002747, -0.38551998138427734, -0.5262355208396912, 0.057666804641485214, 0.9993013143539429, 0.5158967971801758, 0.6620052456855774, 1.620764970779419, -0.5411196947097778, -0.7272599935531616, 2.202470302581787, 1.0682132244110107, 0.32668715715408325, 1.6246216297149658, -0.8508651852607727, 0.5504477024078369, 0.02543172612786293, -0.7052329778671265, 0.8961493372917175, -0.6351247429847717, -0.9484106302261353, 0.20072954893112183, -0.3943384885787964, -0.4743227958679199, 1.0667011737823486, 0.6297725439071655, 0.3447096347808838, 0.05193649232387543, -0.5218712091445923, 0.1732059121131897, -0.20714418590068817, -0.03438045457005501, 0.1223592758178711, 1.0160322189331055, -0.7326546907424927, 1.433976650238037, 0.19189438223838806, -0.07764910906553268, 0.37643328309059143, -0.5052270293235779, -1.4584989547729492, -1.0983047485351562, 0.6391230225563049, -1.8265392780303955, -2.2313332557678223, 1.0302340984344482, 0.20354866981506348, -0.5171112418174744, 0.32228440046310425, -1.4377152919769287, 0.6390239596366882, 0.5884692668914795, 1.4333502054214478, -0.39221444725990295, 0.9108864665031433, -0.10088874399662018, 0.20020556449890137, -0.06412801146507263, 1.0186147689819336, -0.22141128778457642, 0.9461697936058044, -0.23412853479385376, -0.3944071829319, -0.38897866010665894, 0.5384474992752075, 0.1347547471523285, -0.12487516552209854, -0.5590875744819641, 0.6066842079162598, -1.5078918933868408, 0.178673654794693, -1.3411163091659546, 0.12504953145980835, -0.06993480026721954, 0.7145158648490906, -0.3358103930950165, 0.8337740302085876, 0.4212598502635956, 0.9573742151260376, 0.5238237977027893, -1.3679494857788086, 0.961456835269928, 0.27869319915771484, 0.7691455483436584, -1.1162103414535522, 1.1014962196350098, 0.563772439956665, -0.06377175450325012, 0.6034408807754517, -1.9782286882400513, -1.9334760904312134, 1.1420913934707642, 0.11277344822883606, -0.58369380235672, 0.32736513018608093, 0.6707324385643005, -1.5539367198944092, 0.105471670627594, 1.2365953922271729, -0.1743372529745102, -0.7099594473838806, 0.7882440090179443, -0.05956457927823067, 0.37834954261779785, 1.1241354942321777, -0.16341757774353027, 0.7625629305839539, -0.9934731721878052, 0.08155501633882523, -3.0862209796905518, 0.5331165790557861, 0.30998238921165466, 0.1024397686123848, 1.6651029586791992, 0.11435406655073166, -1.5017822980880737, 1.3017189502716064, -0.303843230009079, -2.7938921451568604, -0.45158883929252625, 0.7358095049858093, -0.8604212999343872, 0.895431399345398, 2.109327793121338, 1.618402361869812, -1.0334266424179077, 0.19135336577892303, 0.8173153400421143, 0.8581381440162659, 0.7194398641586304, -0.5325999855995178, 1.203424096107483, -0.9145625233650208, 0.18100084364414215, -0.2118779569864273, 0.4713061451911926, 0.6532534956932068, 0.265499472618103, 1.4998128414154053, 0.6723527312278748, -2.707811117172241, -0.05326785147190094, 0.9344910979270935, 0.40161028504371643, -0.5367783308029175, 0.7210660576820374, -1.000524878501892, 0.18226288259029388, 0.12909036874771118, -1.3831640481948853, -1.834977149963379, -0.5615838170051575, 1.1448208093643188, -0.4885088801383972, -0.47766321897506714, 0.01667831651866436, -0.9780059456825256, -0.7495042681694031, 0.7362810373306274, -1.0600892305374146, -0.7466323375701904, 0.1254929155111313, -0.8379519581794739, 0.4570518732070923, 0.37988361716270447, 1.5654301643371582, -0.10887718945741653, 1.426296353340149, -0.5593205094337463, 0.8654972314834595, 1.563173770904541, 1.0472005605697632, 0.5714378356933594, -2.0990283489227295, -0.37726902961730957, 0.17500954866409302, 0.2397983968257904, 0.03061877377331257, 0.6474476456642151, 0.5077557563781738, -1.1641557216644287, 0.18120254576206207, 1.5243480205535889, -0.7128302454948425, -0.19104741513729095, -0.37233179807662964, 0.08600932359695435, 0.3090268671512604, -0.6660106778144836, 0.18835188448429108, 0.8367598652839661, 0.47435078024864197, -0.6641726493835449, -1.2189916372299194, -0.01646055094897747, 1.4206241369247437, 1.2665992975234985, -0.9764286875724792, -0.9723937511444092, 0.4991282522678375, 0.5397341847419739, 1.509639024734497, 0.8312332034111023, 1.2234580516815186, 1.1103570461273193, -0.48200535774230957, -1.0261305570602417, 0.9734923243522644, -0.646343469619751, -0.9130582809448242, -0.2997404932975769, -0.8961582183837891, -0.5088436007499695, 1.1675156354904175, -0.509032130241394, 0.03262167051434517, -0.7626100182533264, -0.3962610363960266, 0.777378261089325, 0.5215631127357483, 0.12146763503551483, 0.9141205549240112, -0.6940453052520752, -0.6261299252510071, -0.3583708107471466, -0.3701675236225128, 0.304151713848114, -0.6774182915687561, -0.39163070917129517, 0.8477038741111755, 1.271657109260559, -1.0983539819717407, -1.5240552425384521, -0.22863653302192688, 0.02689146250486374, 1.077931523323059, -0.1232401579618454, 0.04843810200691223, 0.2538434863090515, -0.4568016231060028, -0.001392801059409976, 0.8350300192832947, -0.44846731424331665, 1.288698434829712, -0.7991214990615845, 1.3940753936767578, 1.155846357345581, 0.8680094480514526, 0.8157707452774048, -1.9300613403320312, 1.6295353174209595, -1.6895262002944946, -2.1105329990386963, 1.205089807510376, -0.9404239654541016, -2.212341070175171, -1.056594729423523, 0.17968037724494934, -0.5347439050674438, 0.30558156967163086, 0.2934912145137787, -0.5107951164245605, 0.7673879265785217, -0.34425872564315796, -0.55572909116745, -0.5129072070121765, -0.47879090905189514, -1.5735876560211182, 0.3105742037296295, 0.4249323606491089, -1.1771730184555054, -0.3330990672111511, -0.7951666712760925, 0.6051169037818909, -0.5512523055076599, -0.1658884733915329, 1.9277489185333252, -1.8691092729568481, 0.448710560798645, -0.3952249586582184, -0.6068940162658691, 1.1797009706497192, -2.461073160171509, 0.12205459922552109, -1.265118956565857, 1.000670313835144, 0.21577100455760956, -0.3415515124797821, 1.197139859199524, -0.47929081320762634, 1.8210489749908447, -2.3271963596343994, 0.7158942818641663, 0.6406159996986389, 0.8720168471336365, -0.5633744597434998, -0.2510167956352234, 0.5986231565475464, -0.19496110081672668, -0.2766974866390228, -0.746724009513855, -0.25892502069473267, -0.6977033615112305, -1.36075758934021, -1.060442328453064, -0.9763632416725159, 0.3820149600505829, -0.30788305401802063, 0.5094213485717773, 0.19220872223377228, 1.3410801887512207, 0.7989577651023865, 0.26661646366119385, 0.4279547929763794, -0.3987954556941986, -0.7982105016708374, -0.8910865783691406, -0.13657677173614502, -0.4738614857196808, -2.1340596675872803, -0.42452356219291687, 0.5813318490982056, 0.3506518304347992, -0.168922558426857, -1.7285183668136597, 0.1812741756439209, 0.9292439222335815, 0.5844888091087341, 2.4880034923553467, -1.2192384004592896, 2.352587938308716, 0.8902055621147156, 0.8523717522621155, 1.3683557510375977, 0.18030612170696259, 0.11480624973773956, 1.563050627708435, 0.5352959036827087, 0.1067110076546669, -1.5037357807159424, 0.731415331363678, 0.418469101190567, 0.012446076609194279, 0.3271729052066803, -0.6892409324645996, 0.06202400103211403, -0.4379473924636841, -0.36942699551582336, 1.038324236869812, -0.3146268129348755, 0.10793076455593109, 2.4758267402648926, -1.0810127258300781, -0.776442289352417, -0.18185685575008392, -0.25303566455841064, -0.7713913321495056, 0.6614611744880676, -1.2125357389450073, 0.4584047794342041, 0.3615660071372986, -1.2535700798034668, 0.09289287775754929, -0.2609030604362488, 0.5293205976486206, 0.9083951711654663, -0.7929164171218872, -0.7426341772079468, -1.3571593761444092, 1.114696741104126, -0.6584665775299072, 0.15956176817417145, 1.1370515823364258, -1.172119379043579, 0.8915748596191406, -0.025357741862535477, 0.3287438750267029, 0.5714008212089539, 1.3573970794677734, -0.19077923893928528, -0.7116252779960632, 0.9587106704711914, -0.23968610167503357, -0.3585016429424286, -0.9188433885574341, 0.28710460662841797, 2.008622884750366, 1.5150575637817383, 2.3479268550872803, -1.5201338529586792, -0.10235242545604706, -1.7708717584609985, -0.21987886726856232, -1.3816510438919067, -0.7722469568252563, -0.06647773832082748, 0.5013722777366638, 0.8443801403045654, -0.8699097037315369, -0.18930891156196594, 0.34392350912094116, -0.9973589777946472, 0.09540119022130966, -1.0747413635253906, 0.5935581922531128, -0.37363412976264954, -0.22362039983272552, 1.3245832920074463, -0.5288355350494385, 0.047261793166399, 0.13997802138328552, 1.5411628484725952, 0.6029447913169861, 0.7697228789329529, 0.36325666308403015, 0.495186448097229, 0.022130997851490974, -1.7325222492218018, 1.2518672943115234, 1.7698948383331299, -0.7900750041007996, 1.133958101272583, -1.0381858348846436, 0.23446457087993622, 0.2317616194486618, -1.4903199672698975, -0.37020614743232727, 1.1132287979125977, -1.199429988861084, -0.7648976445198059, 0.0635673925280571, -1.368801236152649, 1.5682144165039062, 0.17551717162132263, -0.08245088160037994, -0.6434643864631653, 0.7110428810119629, -0.6965611577033997, 0.41967737674713135, 0.15087057650089264, 0.3004038333892822, -0.5635048747062683, 1.4872486591339111, 0.1751995086669922, 0.5045689940452576, 0.2613644599914551, -0.6926335692405701, -0.17847609519958496, 0.08425809442996979, 0.5263162851333618, 0.10607953369617462, 0.17344284057617188, -0.19942384958267212, -0.7336699366569519, 0.9548165798187256, 0.24653789401054382, -1.2087558507919312, 1.726640224456787, -0.7831096649169922, 0.6997089385986328, 0.6505776643753052, 0.15216411650180817, -1.0520505905151367, 1.6682484149932861, 1.4384915828704834, -0.16212044656276703, -1.2725039720535278, 0.6424321532249451, -0.29551249742507935, 1.3074079751968384, 0.7142311334609985, 1.044499397277832, 0.453388512134552, 0.5218233466148376, -2.0204975605010986, -0.5528259873390198, 1.1903223991394043, -0.9502888917922974, -0.1906866878271103, -1.5092684030532837, -0.9856336116790771, 0.3673991560935974, -1.169192910194397, -0.41842812299728394, 1.1009327173233032, 0.7629207968711853, 1.333267092704773, 0.8546820878982544, 1.2723979949951172, -0.48509490489959717, 0.4677915573120117, 1.3434240818023682, 0.7376269102096558, -0.8560139536857605, -0.4239928126335144, -0.05029081925749779, -0.687542200088501, -0.13099128007888794, 0.30452191829681396, 0.6652116775512695, 2.1099958419799805, 1.1996990442276, 0.48444443941116333, -1.3436387777328491, -1.320884108543396, -0.31447702646255493, 0.33990538120269775, 1.292198657989502, -0.8177909851074219, 2.2277045249938965, 0.24365898966789246, -0.01436623651534319, 1.0325970649719238, 0.07793228328227997, 1.049698829650879, -0.9717459678649902, 0.47596457600593567, -0.37400829792022705, 1.2060716152191162, 0.21043680608272552, -1.2213444709777832, -1.4360036849975586, 1.7915934324264526, 0.2759511172771454, 1.0154958963394165, 1.6044448614120483, 0.6015008091926575, 0.14760781824588776, -0.17755848169326782, 1.2272816896438599, 1.782639980316162, -0.19945235550403595, -0.33212924003601074, -0.4878332018852234, 0.5641544461250305, -0.6441808342933655, 0.5143316984176636, -0.040459223091602325, 0.11746319383382797, -0.8978874683380127, -1.9975759983062744, -0.5054157376289368, 1.9308178424835205, -1.6294405460357666, -0.13679108023643494, 1.0806719064712524, 1.4833358526229858, 0.21646232903003693, -0.09421271830797195, -0.444834440946579, -0.30113816261291504, 0.6458572149276733, -0.8921051621437073, -0.30182358622550964, 1.3571910858154297, -0.12704545259475708, -1.1812118291854858, 0.42542940378189087, -0.2815517783164978, -0.71799635887146, -0.235918790102005, 0.9813835620880127, 0.39992576837539673, 0.446634978055954, -2.0707509517669678, 0.7583167552947998, -1.3543462753295898, 0.04315317049622536, -0.1849879026412964, 1.2592270374298096, -0.002184015465900302, 0.26830917596817017, -0.5885629057884216, -0.7147296667098999, -1.29878568649292, 0.7893239855766296, 0.3054504096508026, -0.6090885400772095, 1.43450129032135, 1.2297248840332031, 2.024972438812256, 0.11649484932422638, 0.00924478005617857, -0.690559983253479, 0.29372018575668335, -0.030804071575403214, 0.24097007513046265, -0.02629815973341465, -0.38872233033180237, -0.8701569437980652, -0.8309357166290283, -0.961208701133728, -0.15748153626918793, -0.9960331320762634, 0.8671997785568237, 1.1707450151443481, 1.1270447969436646, -0.05713943392038345, -0.7516078948974609, 0.4915209114551544, 1.0805776119232178, -0.014264950528740883, 0.6150687336921692, 0.09679736196994781, -0.1116531565785408, -1.541979432106018, -0.9165070056915283, -1.6657158136367798, 0.7575139403343201, -1.944040298461914, -1.0909221172332764, 1.2698214054107666, 0.1766035258769989, -0.18502242863178253, -0.3066999912261963, 0.4449951648712158, 0.5634360909461975, -1.735748529434204, -0.5541900396347046, 1.284558892250061, 0.45980820059776306, 1.6250441074371338, 0.3815931975841522, 0.5733026266098022, -1.4992170333862305, 0.27766019105911255, 0.34202662110328674, -0.4714295268058777, 1.8705631494522095, -0.6460505127906799, 0.34008488059043884, -0.3720322251319885, -0.289677232503891, -0.3535604178905487, -0.0430072620511055, -0.9358395934104919, 0.03927832841873169, -0.865537703037262, 0.06400073319673538, -0.5346444845199585, -0.517544686794281, 0.8143377900123596, -0.16712376475334167, 0.13359127938747406, -0.2758537530899048, -0.5476633906364441, 1.381308913230896, 0.4811738133430481, -0.8748311400413513, 0.831213653087616, -0.130461648106575, 0.8149521350860596, -0.5783067941665649, 2.227073907852173, 0.48432886600494385, 1.6020195484161377, -0.5553789138793945, -1.0764176845550537, 0.9687822461128235, -0.5806766152381897, -0.6567445993423462, -1.4004417657852173, -1.1626332998275757, 0.4116517901420593, -1.036161184310913, 0.26483529806137085, -1.3247478008270264, -1.2007606029510498, 1.3821769952774048, -0.38969898223876953, -1.3445959091186523, 0.011917936615645885, 1.2522175312042236, 0.9906135201454163, -0.6897609829902649, -0.45782721042633057, 0.18616408109664917, 2.3501391410827637, 0.6187211275100708, -1.946257472038269, 2.095559597015381, 0.6581827998161316, 2.097881555557251, -0.14738722145557404, -2.6763501167297363, -0.007863983511924744, 0.792914092540741, 0.5007578730583191, -0.8061358332633972, 0.7473189830780029, 0.5331956148147583, -1.255141258239746, -0.390613317489624, -0.7048166394233704, -0.05120234936475754, -0.6864202618598938, 0.7050586938858032, 0.6543041467666626, 1.8697201013565063, -0.36619341373443604, 2.622302293777466, 0.7871577143669128, 0.37612679600715637, 1.267772912979126, 0.14006945490837097, 0.439512699842453, 1.7730755805969238, 0.2734437584877014, 0.2604619264602661, -0.46435925364494324, -0.8800787329673767, 0.6374592185020447, -0.9328824281692505, -2.0456061363220215, -0.3467344045639038, -0.6664662957191467, -2.03123140335083, -1.6115171909332275, -0.4679337739944458, -0.2543014883995056, -1.2782360315322876, 0.23170259594917297, -0.6161170601844788, 0.5919737219810486, -0.1419011652469635, 0.4025006890296936, -0.1313512921333313, 1.3498605489730835, 0.5567807555198669, 2.4649295806884766, 1.514297604560852, 0.40415066480636597, 0.24169205129146576, -1.1329344511032104, -0.39535409212112427, 0.6585833430290222, -0.5476659536361694, 0.12190251797437668, 1.2231979370117188, 0.24819797277450562, 2.367417812347412, 0.9001971483230591, -0.5475232005119324, -0.38536033034324646, 2.1502113342285156, -0.027288692072033882, 0.38660329580307007, 0.39527857303619385, 1.353259563446045, -0.25801411271095276, -0.6898580193519592, 0.8838318586349487, 0.547599196434021, -0.5193711519241333, -0.8292229175567627, -0.4391983449459076, 0.13393715023994446, -0.10100936889648438, 0.9386611580848694, 0.7717449069023132, 1.524877905845642, -1.3906522989273071, -0.08472529798746109, -1.9609073400497437, 0.8299301266670227, 0.08486734330654144, 0.07543186843395233, 1.011874794960022, -0.5553227066993713, -0.5451680421829224, 2.1449804306030273, -1.1638141870498657, 1.7328823804855347, -1.3289682865142822, -0.2608242630958557, 1.7968902587890625, 1.2715468406677246, -0.11694587767124176, -0.5485990047454834, 0.338976114988327, 0.11815763264894485, -0.7376855611801147, -0.24163798987865448, 0.6601958274841309, 2.2022409439086914, -0.2650967538356781, -1.4510577917099, -0.6021369695663452, 0.8102719187736511, 0.07068132609128952, -0.9614217877388, 0.685741126537323, -0.031194856390357018, -0.17484594881534576, 0.4759725034236908, -0.3709544241428375, -0.3227119743824005, -0.35917767882347107, -1.1951313018798828, 0.4321872889995575, -1.4359235763549805, 0.0682365670800209, -0.1708267629146576, -1.0794366598129272, 1.242549180984497, 0.36778315901756287, -0.8910694718360901, 0.45568108558654785, 0.27840083837509155, 1.0707100629806519, -0.006696690339595079, -1.4186830520629883, -0.3948146402835846, 0.23027756810188293, 0.6208392977714539, -0.7300115823745728, -1.3838764429092407, 1.333542823791504, 1.6557697057724, 0.18774797022342682, -0.679580807685852, -1.753973126411438, 0.5259721279144287, 0.2507977783679962, 0.4430026710033417, -2.3050012588500977, -1.4856144189834595, -0.8892635703086853, -1.1693627834320068, 1.345464825630188, 1.5101312398910522, 0.052282992750406265, -1.1662354469299316, -0.29628676176071167, -0.17631793022155762, 1.0135551691055298, -0.732110857963562, 0.11441615968942642, 0.06930192559957504, 0.02673153765499592, -0.18222373723983765, 0.5410658717155457, 0.6672084927558899, -0.4136543869972229, 0.03857821226119995, 0.7016990184783936, 0.3618583679199219, 0.4867463707923889, -2.624358892440796, -1.5038148164749146, 0.1503073275089264, 0.9406060576438904, 0.4095168113708496, -1.204996943473816, -0.20092220604419708, 0.8441566824913025, 0.946860671043396, 1.0501805543899536, 0.3174293339252472, 0.42429423332214355, -0.3809639513492584, -1.0690224170684814, -1.1419363021850586, -0.32946887612342834, 1.0172615051269531, 0.03540932387113571, 1.536779761314392, 0.931699275970459, -1.1568233966827393, -1.399019718170166, 0.5199300050735474, -0.8016870021820068, -0.9473179578781128, -0.666612446308136, 0.8488914966583252, 0.8692481517791748, -0.8375146985054016, 0.2775258421897888, -0.6786684393882751, -0.7900788187980652, -0.1359451711177826, 1.3459917306900024, -0.27205368876457214, -0.6649086475372314, -0.5995043516159058, 0.8845372796058655, 2.3835010528564453, 1.579363226890564, -0.9184269309043884, 0.01778508350253105, 1.9620496034622192, -1.3705205917358398, -0.7521467208862305, -1.433586597442627, -0.3347057104110718, -0.37588220834732056, -1.5800936222076416, -1.865470290184021, -0.19926244020462036, -0.10758472979068756, 0.8888034224510193, 0.07775403559207916, -1.2535619735717773, -0.8131898045539856, 0.9940564632415771, -0.3699522018432617, 0.8822917342185974, -0.17927691340446472, -0.010054348036646843, -1.4783722162246704, -1.4745421409606934, -1.112052321434021, -0.8981764912605286, -0.25819799304008484, 0.5729352235794067, -0.8356552720069885, -0.14113861322402954, 0.1978338211774826, -1.3415628671646118, -0.7476699948310852, -0.9526900053024292, 0.4276069700717926, -0.5329502820968628, -0.009287796914577484, 0.9493535161018372, 0.8672302961349487, -0.6980782747268677, 1.0287119150161743, -1.5498625040054321, -0.08752100169658661, -0.6402877569198608, 0.6599844694137573, -1.4386461973190308, -1.5633585453033447, 1.3334623575210571, -0.3283710777759552, 0.44446611404418945, 0.1549903154373169, 0.6541483998298645, -1.520567536354065, 0.001553271315060556, -0.22216840088367462, 0.1767478883266449, -0.32901471853256226, -0.2954707145690918, -0.33396443724632263, 0.5639501214027405, 0.6296191811561584, -1.0082377195358276, 0.43804529309272766, 0.5459055304527283, -1.8001006841659546, -0.012289277277886868, -0.8576620817184448, 0.8811196684837341, 0.9522926211357117, 0.6504977941513062, -0.28514590859413147, 1.9247945547103882, -1.6166759729385376, -0.16595982015132904, 0.20200102031230927, 0.45408400893211365, -0.08657406270503998, -1.2381479740142822, 0.9945324659347534, -1.750452995300293, -0.7151128649711609, 1.3752071857452393, -0.29083216190338135, -0.5565659999847412, 1.3425861597061157, 1.1802414655685425, 1.6021788120269775, -0.5966907143592834, 1.1682013273239136, -1.6541461944580078, 0.331813246011734, 1.9522442817687988, 1.0912758111953735, 0.5643249750137329, 1.0882470607757568, -0.05080171301960945, -1.4277979135513306, -0.9967475533485413, -1.0565966367721558, 0.35192182660102844, 0.25153082609176636, 0.6859855055809021, 1.877865195274353, -0.6179061532020569, -0.9609321355819702, 0.5622830390930176, -0.8718848824501038, -1.3719509840011597, 0.08415765315294266], [0.47974884510040283, 0.210798442363739, 0.4015430510044098, -0.15393272042274475, 0.08312160521745682, 1.5446841716766357, -1.1963257789611816, -2.0774037837982178, 1.3483705520629883, -0.34201857447624207, -1.2531428337097168, 0.16144435107707977, 0.4298960566520691, 1.560072422027588, 0.9921465516090393, -0.19672927260398865, -0.8024556636810303, 0.6404324173927307, -0.29564812779426575, -1.0258859395980835, 0.18799255788326263, 1.8428446054458618, 0.9454206824302673, 1.8620316982269287, -0.43768131732940674, 0.8523223996162415, -1.316019892692566, -1.2579114437103271, -0.11034596711397171, 2.419127941131592, 2.2878150939941406, -0.5914382338523865, -1.5681569576263428, -0.04441332072019577, -0.8640983700752258, 1.0196093320846558, -0.6051096320152283, 0.3072402775287628, 0.6518893837928772, 0.9742917418479919, -2.250976800918579, -0.3019327223300934, 0.5065127015113831, -0.36036697030067444, -0.1491711437702179, -1.1869380474090576, -0.2653801441192627, -1.1429429054260254, 0.587085485458374, -1.2500585317611694, -0.03027402237057686, 0.14287592470645905, -2.32658314704895, -1.8312371969223022, 1.0153520107269287, 1.9660875797271729, -1.7282317876815796, 0.4731392562389374, 0.23319905996322632, 0.2531389892101288, -1.045088529586792, 0.9365895390510559, 0.5276853442192078, 1.2494676113128662, 0.6813989281654358, 0.72910076379776, 1.5324848890304565, 0.26711326837539673, 1.535486102104187, -1.6230037212371826, -0.6392199993133545, 0.7350888252258301, 0.40451931953430176, -0.5128437280654907, 0.46741586923599243, -0.9772804975509644, -0.5302038788795471, -0.23055677115917206, -0.7999777793884277, 0.972691535949707, -1.1903033256530762, -0.7696181535720825, 0.4189580976963043, 0.11064007133245468, 0.6573065519332886, -1.9541720151901245, 0.7656341195106506, 1.09234619140625, -0.12518155574798584, 0.48930519819259644, 0.42121681571006775, 0.4582056403160095, 0.6903824806213379, 0.9274516105651855, 1.469197392463684, -1.0619702339172363, 0.6385829448699951, -0.449799507856369, 0.1570502370595932, -1.1465107202529907, 1.012478232383728, -0.257562518119812, -0.14265607297420502, 0.1481204628944397, 0.8043832778930664, -0.5250144600868225, 1.1584078073501587, 0.8027623891830444, -0.6090220212936401, -0.09016434103250504, 0.6615562438964844, -1.161346673965454, -0.16140568256378174, 2.0441465377807617, 0.44051721692085266, 0.5366809368133545, -0.924440324306488, 0.27276065945625305, 0.2607830762863159, -0.5894737839698792, 0.061683304607868195, -1.7416800260543823, 0.7274256944656372, -0.6994690895080566, 0.21637822687625885, 0.36835330724716187, -1.3847285509109497, -1.024873971939087, -0.9501305222511292, -1.1543956995010376, -0.31332457065582275, -0.7983841300010681, -0.26966309547424316, 0.7582623362541199, 0.8680054545402527, 0.4668465554714203, -0.3788596987724304, 0.42489001154899597, -1.2898623943328857, -0.6807897686958313, -0.36209917068481445, 1.5550565719604492, -0.0906819999217987, 0.38797569274902344, 0.2630710005760193, -0.44419559836387634, -0.6009276509284973, -0.40308085083961487, 1.166549801826477, -0.4174859821796417, -0.5126559734344482, 0.8907605409622192, -0.103511281311512, 0.15987840294837952, -0.7729976177215576, -0.6374990940093994, 0.9799309372901917, -0.8854434490203857, 0.22578586637973785, 0.3043694496154785, 0.7684637308120728, -0.2612714171409607, -0.7141936421394348, -1.5822421312332153, 0.3081390857696533, -1.083162784576416, 1.2247974872589111, -1.2545051574707031, -0.8922601342201233, -1.3481303453445435, 0.32271790504455566, 1.0072910785675049, 0.48922157287597656, -1.453249454498291, -0.22629262506961823, 2.576968193054199, 0.8765631914138794, -0.1629205197095871, -0.4060070514678955, 1.4561183452606201, 1.5364607572555542, 1.0157545804977417, 1.373580813407898, 0.6148259043693542, 1.8872859477996826, -0.2309054285287857, 0.8180397748947144, 0.32430386543273926, 1.1352404356002808, 0.19731707870960236, -2.2212069034576416, 1.3155313730239868, -0.0428953655064106, 0.9091832637786865, -0.7372239828109741, -0.46532732248306274, 0.4197637140750885, -0.14400064945220947, -0.8153464794158936, -0.8124122023582458, 0.15125730633735657, -0.45063304901123047, 0.6366608738899231, -0.040335558354854584, -0.4783710539340973, 0.6602990627288818, -1.3556512594223022, 0.10402514785528183, -0.0009018790442496538, -0.4975382387638092, -0.9937033653259277, -0.40451642870903015, 0.9950577616691589, 1.3628896474838257, 0.37884432077407837, -0.8757539987564087, -0.11118628084659576, 1.5059067010879517, -0.8481602668762207, 0.2902018129825592, 0.5569713711738586, 1.0258005857467651, -0.30842965841293335, -0.2470763921737671, -0.19418703019618988, 0.9916592240333557, 0.48985597491264343, -0.03798345848917961, 2.541820526123047, 0.2977044880390167, -0.010055097751319408, -0.7220630645751953, -0.08983739465475082, 0.37607496976852417, -0.8808305263519287, -1.12423574924469, -1.467645287513733, 0.2662530243396759, -0.9922543168067932, 1.1772396564483643, 0.39523810148239136, -0.23241294920444489, 0.37013763189315796, 1.4700437784194946, -1.0523245334625244, -1.7969282865524292, 0.6695127487182617, 0.9366409778594971, -0.16413159668445587, 0.5234840512275696, -0.5868339538574219, -0.9528067111968994, -1.32784903049469, 0.14697779715061188, -0.3013816475868225, 1.8671878576278687, -0.4484405219554901, -0.16076155006885529, 0.5440652966499329, -1.221112847328186, -0.5787470936775208, 0.6652277112007141, 1.3919428586959839, 0.29081735014915466, 0.7392452955245972, 0.44459259510040283, -1.1079963445663452, -0.8565544486045837, -0.5998493432998657, -1.9171741008758545, -0.8320187926292419, -1.4201401472091675, -1.4415698051452637, 1.8076945543289185, -0.9192887544631958, 2.148958206176758, 0.33603137731552124, -0.47894883155822754, -0.38508057594299316, 1.4730548858642578, -0.5633869171142578, -0.9776445031166077, -1.4290237426757812, -0.49035319685935974, 0.9139875173568726, 0.2948700487613678, -1.0368964672088623, 1.6850799322128296, 0.8661934733390808, 3.0073471069335938, -0.7787709832191467, 0.2938738465309143, 0.21737787127494812, -0.6253116130828857, 1.0123698711395264, 0.5019674301147461, 0.11081234365701675, -0.08710313588380814, 1.8590025901794434, -0.7215836048126221, 2.2814695835113525, -0.4236234426498413, 0.24492518603801727, 1.8377059698104858, 0.5801280736923218, -0.7492496967315674, 0.47772011160850525, -2.5596365928649902, 0.21346405148506165, -0.81895911693573, -0.5011330842971802, -0.6278643608093262, 0.6654140949249268, -0.5036109089851379, 0.9887740612030029, -0.45675918459892273, -1.089626431465149, -0.12593252956867218, -1.206505298614502, -0.5086164474487305, -1.2363717555999756, 0.5237475633621216, -0.05814068019390106, -0.5654667615890503, -0.6943292617797852, -0.1867889165878296, -0.25767603516578674, -1.1975444555282593, 0.867894172668457, 0.871157705783844, 0.44105038046836853, 0.155199334025383, -0.5721270442008972, 0.46847641468048096, 0.26827794313430786, -1.711016058921814, -0.9402697682380676, -1.3455427885055542, -0.13532090187072754, -0.053239960223436356, 0.5788888931274414, -0.07764320075511932, -1.3619791269302368, -0.28288981318473816, 0.4737449586391449, 0.5085278153419495, 0.16557106375694275, -1.1780829429626465, 0.9097957015037537, -0.37339240312576294, 0.825772225856781, -0.02607160247862339, 1.9464737176895142, 1.3199095726013184, -0.571035623550415, -0.6439211964607239, -0.7374190092086792, 0.34681251645088196, -1.951613187789917, 0.647773265838623, -0.1579991579055786, 0.26986220479011536, 1.2014368772506714, -0.04346756264567375, 0.4738825261592865, -1.0086066722869873, 0.9268156886100769, -1.309082269668579, 0.8044814467430115, -0.045010149478912354, -0.9902200102806091, 0.03359381854534149, -0.6800332069396973, -0.15488457679748535, -1.5216867923736572, 0.8977898359298706, 0.8954100012779236, -1.9702787399291992, 0.6686959266662598, 1.8314279317855835, -1.2066937685012817, -0.9816302061080933, 0.21068553626537323, -2.5198974609375, -0.020158221945166588, -1.112635612487793, -0.49897584319114685, 1.2215168476104736, 0.5593575239181519, -0.1781175583600998, -1.9980708360671997, 0.2738531827926636, 1.2158764600753784, -0.8501530885696411, 0.00489046098664403, 2.0198068618774414, 0.6681086421012878, 2.1721839904785156, -0.726576030254364, 2.6485435962677, -1.2440667152404785, -0.9360904097557068, 1.007997989654541, 0.7154931426048279, 1.3726296424865723, 0.17288103699684143, 0.6027359366416931, -1.4778021574020386, -1.946877360343933, -0.4313151240348816, 0.31552034616470337, -1.7712311744689941, 0.0922284945845604, 0.17019011080265045, -0.2876304090023041, 1.1087251901626587, 2.55168080329895, 0.7161604762077332, 0.7494932413101196, 0.8945982456207275, -1.773057460784912, 0.05040919780731201, -0.07999816536903381, 1.0625104904174805, 0.050524938851594925, -1.7602468729019165, 0.8079566359519958, -1.4366817474365234, -0.5608341693878174, -1.298274278640747, 0.8071029186248779, -0.809656023979187, 0.7019959092140198, -1.5008280277252197, 2.2534329891204834, -0.4607071578502655, 1.8840463161468506, -0.5000417232513428, -0.44395774602890015, -1.408401608467102, -0.40948063135147095, 0.1927967220544815, -0.5789502263069153, -0.4786355197429657, 1.8078159093856812, -2.0570144653320312, -0.11684487015008926, -0.705304741859436, -1.3497449159622192, 1.118629813194275, -1.0871649980545044, 2.557264566421509, 0.717660665512085, 0.8991162180900574, -0.4021513760089874, -0.6195187568664551, 0.09480655938386917, -1.0241249799728394, 1.133962869644165, 1.464870810508728, -0.4558482766151428, -1.117379069328308, 0.5455710887908936, 0.08831602334976196, -1.822723627090454, 0.07595553994178772, -0.016890913248062134, 0.025315767154097557, 0.9651418924331665, 0.30214568972587585, -0.5266122817993164, 0.6925613284111023, 0.1412763148546219, -0.5135359764099121, -0.4220295250415802, 0.2710249423980713, -0.25792911648750305, -1.3901710510253906, 0.057225171476602554, -0.613548219203949, -7.846106018405408e-05, 0.16094549000263214, 1.1213252544403076, -0.06558442115783691, 1.5658501386642456, -0.43921056389808655, 0.624568521976471, 2.7026519775390625, -0.7344166040420532, 1.2244465351104736, -0.48548394441604614, 2.697016716003418, -0.012480011209845543, -0.7315345406532288, -1.0399353504180908, 1.6963163614273071, 1.0193257331848145, 0.9922888875007629, 1.8825656175613403, -1.10207998752594, 0.26520875096321106, -0.018093718215823174, 0.4823298752307892, 0.5870260000228882, -2.234221935272217, 0.5902584791183472, -0.1618906557559967, -0.6316088438034058, -0.2688027024269104, 0.1840771734714508, 1.140153408050537, -1.2416754961013794, 0.24406324326992035, -0.31500935554504395, -0.8346671462059021, 1.4072242975234985, -0.9770967364311218, 0.9818258881568909, -0.5336584448814392, 0.11793795973062515, 1.1120854616165161, -0.6926829814910889, 2.443856716156006, 1.058504343032837, -1.453499674797058, -1.3731389045715332, 0.726518452167511, -0.32058173418045044, -3.078369617462158, 1.1269516944885254, 0.38004162907600403, 1.10395085811615, 0.4054001569747925, -0.685574471950531, -0.21096433699131012, -0.22806251049041748, 0.5955796241760254, -0.12138308584690094, -0.34956803917884827, -1.2177796363830566, 0.8520086407661438, 1.3597019910812378, 0.4169332981109619, 1.1153123378753662, 1.6533892154693604, 0.5046173930168152, -1.8827351331710815, -0.3322894871234894, 0.6576732993125916, -0.9967579245567322, 1.4462025165557861, -0.008799288421869278, -0.4387050271034241, -2.0347697734832764, -1.198107361793518, -0.08577646315097809, -1.2746036052703857, 0.24533559381961823, -0.47912338376045227, 0.23968464136123657, -1.3201736211776733, 0.08105689287185669, 0.6534644365310669, -0.9084749221801758, -0.1472432166337967, 0.08074643462896347, 1.0431371927261353, 0.26656410098075867, 0.012452716939151287, -1.7228270769119263, 2.0609989166259766, -0.08467404544353485, 0.2306017130613327, -0.10072514414787292, -0.4426703453063965, -1.6326063871383667, 1.2934329509735107, -1.4104708433151245, -0.16439896821975708, 0.23621875047683716, 1.5705127716064453, -0.07081829011440277, -0.5417327880859375, -0.9322682619094849, 1.0793108940124512, 1.0646737813949585, 0.8970498442649841, 0.18780173361301422, 1.1961934566497803, -0.7475598454475403, 0.6635525226593018, 0.5167094469070435, -0.4697476625442505, -3.2022855281829834, 1.0796928405761719, -1.020883560180664, 0.12371743470430374, 1.1683870553970337, 0.3554137051105499, 0.4156111180782318, 0.43951719999313354, 0.20944708585739136, -0.3980421721935272, -0.5341224670410156, -1.6979609727859497, -1.1879758834838867, 1.067874550819397, -0.7149996757507324, 0.5708038806915283, 0.7454707622528076, -0.3193862736225128, 1.4237538576126099, -1.6592307090759277, 0.18597181141376495, -0.525484025478363, -0.547631025314331, -1.0809484720230103, -0.018932906910777092, -1.5541269779205322, 0.472459077835083, -0.6431834697723389, -0.7022454142570496, -0.8946335911750793, 2.494933843612671, 0.6715357303619385, -1.3431240320205688, -1.7141722440719604, 1.2753583192825317, -0.01638656109571457, -2.9770359992980957, 2.1611289978027344, -0.7629867792129517, -1.009048581123352, -0.7115933299064636, -0.7903847098350525, 0.6607978940010071, -0.6183781623840332, -1.3091599941253662, -1.0195571184158325, -2.783050537109375, 0.22941236197948456, 0.4291473925113678, 0.2548512816429138, -0.5879342555999756, 0.6608959436416626, 0.6628357172012329, -0.030752141028642654, -1.2064143419265747, 1.4424413442611694, -0.6431860327720642, 0.5386048555374146, 0.16837376356124878, -0.10374121367931366, 0.5071314573287964, -1.1830875873565674, -1.135523796081543, -0.5447519421577454, 1.5412362813949585, 0.9181715846061707, 0.7647514939308167, -0.10468071699142456, 0.33932289481163025, -0.061910394579172134, -1.1801207065582275, -0.26749861240386963, 0.18206553161144257, 0.8361095190048218, 0.16936250030994415, -1.0465121269226074, 0.6028839945793152, -0.7242866158485413, 0.9382328987121582, -0.2603375315666199, 0.814014196395874, 1.7480053901672363, 0.021853551268577576, -1.3813332319259644, 0.9914832711219788, 0.4050273299217224, 1.4584741592407227, 1.1000103950500488, -0.07164296507835388, 0.5901926159858704, 1.6224600076675415, 1.3061753511428833, 0.36034107208251953, 1.2111454010009766, 0.8075045347213745, 0.9541743993759155, 1.0530800819396973, 0.7992121577262878, 1.817969560623169, -0.375241219997406, -1.1939955949783325, 0.324810266494751, 0.26744842529296875, 0.2522290349006653, -1.159096121788025, 1.1736152172088623, 0.7603219151496887, 0.6129186153411865, 0.38771742582321167, -0.8173069953918457, -1.009871006011963, -0.6807186603546143, -0.15166319906711578, -0.5902167558670044, 0.8482613563537598, 0.40667957067489624, 0.1192527636885643, -1.1971532106399536, 0.582947850227356, -0.13559196889400482, -0.8863624930381775, 1.7020068168640137, 1.7204675674438477, -0.9685628414154053, 1.6694897413253784, 2.586268901824951, -1.4027031660079956, 0.4688246548175812, 0.41841739416122437, 2.107318878173828, 0.7480576038360596, -0.7489213943481445, -1.5560170412063599, -0.9693284034729004, -1.7083148956298828, -2.3328793048858643, -0.22122766077518463, -0.12986566126346588, 0.3828549087047577, -1.3837751150131226, 0.6215741038322449, -0.7099882960319519, 0.9259321689605713, 0.2868962585926056, -1.787772536277771, 0.07253009080886841, -0.7314695715904236, 1.1598386764526367, 1.1200894117355347, 0.3474288582801819, -0.5532442331314087, -0.9368001818656921, -0.7086987495422363, 0.6263284683227539, -0.38850846886634827, 0.17199872434139252, 0.44281089305877686, 1.579403042793274, -0.17603832483291626, 1.2108339071273804, -0.03398064151406288, 0.38051772117614746, -0.8004502058029175, 0.4794297218322754, 0.02509707771241665, -0.7458725571632385, -1.3557875156402588, 2.4831936359405518, 1.6752891540527344, 0.9200291633605957, 0.7493444085121155, 1.5220251083374023, -1.1697676181793213, -0.8021558523178101, 2.389869451522827, 1.5140715837478638, -1.5261110067367554, 0.47539642453193665, 1.1474478244781494, 0.2284257560968399, 1.731576919555664, -0.3148411810398102, -0.2323014885187149, 1.2721035480499268, 0.5653523206710815, -0.33391666412353516, -0.1400371789932251, 1.4039686918258667, -0.15596315264701843, -0.10375645011663437, 0.6105610132217407, -1.6691274642944336, 0.910794198513031, 0.5949057936668396, 1.3797603845596313, 1.412248134613037, -1.4947577714920044, 0.054886575788259506, -0.34914520382881165, 1.3134711980819702, 0.7930428981781006, -1.5239304304122925, -0.7794246077537537, 1.0434582233428955, -0.5254626870155334, 0.3453185260295868, -0.3682584762573242, 0.5624563097953796, -0.5200856924057007, 0.1619160771369934, -0.8317497968673706, 0.035631585866212845, -2.293914794921875, -0.1008497104048729, 1.2344814538955688, 0.5420753955841064, 0.7057545185089111, -0.1717240959405899, -0.3921704888343811, 1.6351796388626099, -0.6933503746986389, 0.6784794330596924, -0.3376833200454712, 1.0607435703277588, -0.7753056883811951, -0.28645482659339905, 0.37139976024627686, 0.4124865233898163, 0.5109740495681763, -2.0514302253723145, -1.0531365871429443, 1.2889987230300903, -0.9248360991477966, -0.9672961831092834, -1.8427295684814453, 0.9970465302467346, 0.5073687434196472, -0.3020293712615967, 0.9903725981712341, 1.142144799232483, -0.9935082197189331, 0.7608649730682373, 1.4982160329818726, -1.0264317989349365, -0.7657608389854431, 0.10748843103647232, -1.2081708908081055, 0.14782775938510895, -0.1776912361383438, 1.0911997556686401, 0.06035815551877022, -0.7585165500640869, 0.45217448472976685, 0.012712995521724224, -1.7278801202774048, -0.7883374691009521, -2.0531933307647705, 0.0182063989341259, -0.02424546889960766, 0.5295208692550659, 1.1731581687927246, -0.046775080263614655, -0.7446085214614868, 0.4469149112701416, 1.2653930187225342, 0.5837016105651855, 0.02227248065173626, 0.42913180589675903, 2.705003023147583, -1.1734305620193481, -1.2111693620681763, -0.036342330276966095, -0.27411511540412903, 0.4258008599281311, -0.8842028379440308, -0.28596988320350647, 0.2511940896511078, -0.22049593925476074, 0.5312547087669373, 1.806221604347229, -1.7519400119781494, -1.1922839879989624, -0.6247270107269287, 0.07200980186462402, 0.5574812889099121, -0.3745524287223816, -0.9736452102661133, -1.2924270629882812, 0.261157751083374, -0.9597729444503784, 1.6984443664550781, -1.8441860675811768, 0.06784652173519135, -0.43176519870758057, 0.24991539120674133, -2.620863914489746, 0.9375362992286682, 1.2820011377334595, -1.2131755352020264, -0.028724882751703262, 1.8749773502349854, -0.07528536021709442, -1.5794906616210938, 1.1687712669372559, 0.666870653629303, 2.0127415657043457, 1.2440069913864136, 1.4881254434585571, 0.4712497591972351, -0.17477084696292877, 0.7201389074325562, 1.0523455142974854, -0.9480684995651245, 1.0884191989898682, 0.09065742045640945, -0.38966575264930725, -0.061564765870571136, -1.1655646562576294, 1.5648545026779175, 0.2544111907482147, -0.05699806660413742, 0.16884608566761017, -0.033555060625076294, -1.36870539188385, -0.6928603053092957, -0.754159688949585, -0.4632066786289215, 0.6917767524719238, -0.15775009989738464, -0.19273407757282257, 0.9268460273742676, 0.24633437395095825, -1.8704910278320312, 1.1662003993988037, 2.092829704284668, -0.540202260017395, 0.07546122372150421, 0.7188644409179688, 0.3920425772666931, -1.0941065549850464, 1.3544909954071045, -2.801508665084839, 0.4538114666938782, -1.6355924606323242, 2.5284574031829834, 0.570222020149231, 0.3088648021221161, 0.26787012815475464, -1.0579454898834229, 2.0698132514953613, -0.5866802334785461, 1.7827717065811157, -0.3408830463886261, -0.11706610023975372, 0.36998167634010315, 1.1791534423828125, 1.4244298934936523, -1.4046509265899658, -2.7249600887298584, 0.7012669444084167, -1.2118874788284302, 0.4329511225223541, -1.440731167793274, 2.775214910507202, 1.1803932189941406, 1.0174095630645752, -0.25948575139045715, -1.3484035730361938, -0.02135244943201542, -1.1509555578231812, 0.09175851941108704, 0.017274871468544006, -0.8937943577766418, 0.8130843043327332, 0.44396236538887024, 1.0316449403762817, 0.6677619814872742, 1.1977427005767822, -1.4686152935028076, -1.5484552383422852, -0.7424337267875671, 1.3178733587265015, 1.3201544284820557, -1.2871061563491821, -1.4577898979187012, 0.6183111071586609, -1.732295274734497, 0.2447153925895691, -0.008245795965194702, 0.22984810173511505, 2.2990610599517822, 0.050073377788066864, -0.19283710420131683, 0.33483895659446716, 0.9877225756645203, 1.1641666889190674, 1.199232816696167, -0.12902255356311798, 0.7530821561813354, -0.6256417632102966, 0.20392456650733948, 1.0410927534103394, 1.0924345254898071, -0.1506824791431427, -1.0911836624145508, 1.3023682832717896, -1.7739861011505127], [-0.07137710601091385, 0.34824663400650024, -0.05727677792310715, -0.2055336833000183, -0.7248532772064209, -0.10002130270004272, 0.29761576652526855, 0.12638017535209656, -0.8323171138763428, -1.8585225343704224, 1.2074822187423706, 0.17403824627399445, -0.8608747124671936, 0.9167865514755249, -1.523765206336975, 0.3571750223636627, 1.6463273763656616, 0.06810206919908524, -0.7683138251304626, -0.1808709055185318, -0.03820767626166344, -0.9145323038101196, 0.5333154797554016, 1.7525933980941772, -0.377844899892807, 0.7929255962371826, -0.28909385204315186, 0.6553714871406555, -0.3132489323616028, -0.4684738516807556, 0.7810942530632019, 2.2616090774536133, 2.133136510848999, -1.4464253187179565, -1.2132972478866577, 0.43370160460472107, -0.764747679233551, 0.7937414050102234, 0.622819721698761, -1.7974241971969604, -1.0141955614089966, -0.3038093149662018, 0.32610055804252625, -1.5213522911071777, 1.1607787609100342, -0.5940380692481995, -2.680201530456543, 0.07380639016628265, -1.5678764581680298, -0.29592376947402954, -1.0655189752578735, 0.3244273364543915, 0.939007580280304, 0.12798064947128296, 1.1014710664749146, -0.36532261967658997, 1.023658275604248, -0.11483535170555115, -0.20232270658016205, 0.5205841064453125, 0.984822690486908, 0.09208956360816956, 1.8643642663955688, -0.2704748511314392, -1.9459141492843628, -0.8129228353500366, 1.8167600631713867, -1.7875392436981201, -0.7743508815765381, 0.36203551292419434, -2.490640640258789, 0.07641398906707764, 0.04636968672275543, 0.815811276435852, -0.30030742287635803, 0.8872276544570923, -0.3263097107410431, -0.24652209877967834, -1.5728708505630493, 0.9213816523551941, 0.074625663459301, 0.30484357476234436, 1.6629170179367065, -0.1742953211069107, -1.7312668561935425, 1.7813568115234375, -0.16633707284927368, 0.0543929785490036, -0.8618825674057007, 0.4988061785697937, 0.4337998032569885, 0.42922356724739075, 1.44401216506958, -0.6076090931892395, -1.1420143842697144, -0.9681645035743713, -0.3619355261325836, 0.16359378397464752, 0.2144729197025299, 0.41566890478134155, -0.7495517730712891, 1.6541810035705566, -0.6006458401679993, 0.9641419053077698, 0.41718974709510803, -1.8182036876678467, 0.8243988156318665, -0.0788387581706047, -1.77094304561615, -0.27911999821662903, -1.4968515634536743, 1.1106939315795898, 0.2817613482475281, 0.802215576171875, -1.2981503009796143, -0.33094826340675354, 1.432207703590393, -2.015486240386963, -0.19348815083503723, -0.10972627252340317, 0.11636674404144287, 0.9866084456443787, 0.696118950843811, 0.9714459776878357, 1.491064190864563, -0.1487492173910141, 0.6402400732040405, -1.8335866928100586, 0.17768779397010803, -1.409828543663025, 1.1645946502685547, -1.3365989923477173, -0.548163890838623, 0.8626767992973328, 0.4280346632003784, -0.46601152420043945, 1.9389638900756836, 1.1219244003295898, 1.4434579610824585, 0.32928887009620667, 0.9481980204582214, 0.5343443155288696, -0.10628689080476761, 0.6833717823028564, 0.5669482350349426, 0.03716465085744858, -0.34790924191474915, 0.3300348222255707, -3.196502685546875, -0.7019352912902832, 0.06159818917512894, -0.2706286609172821, -0.5804800391197205, 0.9731296896934509, -1.1943475008010864, -0.7008583545684814, 0.19510726630687714, 1.3260886669158936, 0.8586827516555786, -0.296536386013031, -0.09675568342208862, -0.25335556268692017, 1.861438512802124, 0.37189266085624695, 1.3240736722946167, -0.3018043041229248, 1.2845855951309204, -0.7429193258285522, -1.3769354820251465, 0.10157466679811478, -0.19607862830162048, -1.029529094696045, -0.07477309554815292, -0.03796502575278282, -1.2509853839874268, 0.21484120190143585, 0.9557507038116455, -0.04987480491399765, 0.5952671766281128, 0.9256926774978638, -0.3480014204978943, -0.5352241396903992, -0.26166045665740967, 1.2452812194824219, -0.40208080410957336, -0.3384331166744232, -0.3042929470539093, -0.9341201782226562, 0.09116639196872711, 0.5717398524284363, 0.8129135966300964, 0.3896177113056183, 1.2104673385620117, 2.004805326461792, 0.39590275287628174, 0.5858318209648132, -0.6253830194473267, 1.577873706817627, -2.192438840866089, 0.6932708024978638, -0.7502604126930237, -0.7414932250976562, -1.4872510433197021, -0.3559681475162506, 0.26461952924728394, -1.6832749843597412, 2.8034017086029053, -0.012378831394016743, -0.585349977016449, 0.7324666380882263, 1.049709439277649, 0.6709307432174683, 0.9735376238822937, 2.1328542232513428, -0.6897388696670532, 0.19159992039203644, -0.8407561779022217, 0.49134302139282227, -0.2822202742099762, -0.011174194514751434, -0.3768394887447357, -0.6995022892951965, -0.5769103169441223, 1.0474804639816284, -0.4335671067237854, 1.2714250087738037, -0.7764453887939453, -0.35809168219566345, 1.4540438652038574, -0.37046968936920166, -0.19304434955120087, -0.6094098687171936, 0.5509631633758545, -0.46460583806037903, -0.07242253422737122, 0.48340240120887756, -0.7482748627662659, -0.30509114265441895, 0.32307228446006775, 0.9516965746879578, 1.6937237977981567, -1.310006022453308, 1.3215049505233765, 0.2023991197347641, -0.5851271748542786, 0.7741081118583679, -0.19163185358047485, -1.0475614070892334, 0.3940715789794922, 1.6095176935195923, -0.85239577293396, 1.3268206119537354, 0.008135437965393066, -0.39740148186683655, 1.9342215061187744, 0.16270308196544647, -1.170112133026123, -0.34094884991645813, 0.35562875866889954, -0.2040455937385559, -0.42136988043785095, -1.1899924278259277, 0.935990035533905, -0.052477844059467316, 1.5743387937545776, -0.21383579075336456, 0.4398825764656067, 0.20151950418949127, -0.9847651124000549, 0.020792702212929726, 1.789189338684082, 0.028188595548272133, 0.25901710987091064, 0.9865254163742065, -0.37719079852104187, -1.004103660583496, 0.10524919629096985, 0.9577945470809937, 0.8484869003295898, -0.2840442359447479, 1.0852923393249512, 1.8080055713653564, 1.4348093271255493, -0.8482376337051392, -1.2642993927001953, 0.421646386384964, 1.5077615976333618, -0.3777249753475189, 0.6796161532402039, -0.02643727883696556, 0.6973198056221008, -0.7104772925376892, -1.3136245012283325, 0.26594701409339905, -0.9657119512557983, -0.5843662619590759, 0.5618513226509094, 0.7941001653671265, -1.306203007698059, 2.3575217723846436, 2.3609416484832764, 0.017753086984157562, -0.3070610761642456, 0.27801063656806946, -1.6704109907150269, -1.3580116033554077, 0.8810696005821228, -0.7109937071800232, -0.5816529393196106, 0.5607923865318298, 0.31891193985939026, -0.7535305023193359, -1.882793664932251, -0.28308969736099243, 0.621755838394165, 0.357184499502182, 0.029994627460837364, -0.5785959362983704, 0.4500497877597809, -0.8388895988464355, 0.22193124890327454, 0.4257354736328125, -0.13285085558891296, -3.0418355464935303, 1.1736093759536743, 2.360337018966675, 0.566597044467926, -0.2647554874420166, 0.6020680665969849, -0.5982366800308228, -0.5794736742973328, -0.058002933859825134, -1.0140810012817383, 2.0943524837493896, -0.7093582153320312, 0.487312376499176, -1.4254443645477295, -0.41316133737564087, 0.49522092938423157, -0.850832998752594, -0.8400373458862305, 1.3169678449630737, -0.27996715903282166, 0.6156373620033264, 0.004764675162732601, -1.439106822013855, -0.24486753344535828, -0.1971927285194397, -0.6606119871139526, -0.9257573485374451, -0.01060755830258131, -0.6100982427597046, 2.0675113201141357, 0.6073065996170044, 0.24336622655391693, 1.2921887636184692, 1.0152069330215454, -0.10140710324048996, 0.45428168773651123, -0.38641729950904846, 1.55233895778656, 0.5545068383216858, -1.1591588258743286, 0.09321250766515732, 1.9908051490783691, -1.142918586730957, 0.5396922826766968, 0.8840649724006653, 0.31168460845947266, 2.5439271926879883, 0.356879323720932, 0.694483757019043, 1.0969408750534058, 1.0719027519226074, 1.2373353242874146, 0.927324116230011, 1.6598588228225708, -0.7515889406204224, 0.4399593472480774, 0.8708146214485168, 0.24730776250362396, -1.258329153060913, -0.3804321885108948, 0.2822272777557373, -0.7286317944526672, -0.21828614175319672, 0.19354921579360962, 2.8456242084503174, -1.718788504600525, 1.6647385358810425, -2.14621901512146, 1.6019271612167358, 0.035504523664712906, -1.090461015701294, -0.07809381186962128, -0.06539051979780197, 0.7371306419372559, -1.0858272314071655, 0.6017640233039856, -1.471126914024353, 0.763321578502655, 2.305544376373291, -0.42127394676208496, 2.386597156524658, -0.733775794506073, -0.23523706197738647, -1.3864902257919312, 1.0623854398727417, -1.063468098640442, 0.5488250255584717, 0.6360070705413818, 0.4760001003742218, 0.5621467232704163, 0.9370149374008179, 0.06456263363361359, 0.518369734287262, 0.017138687893748283, -1.4971847534179688, 0.8419511318206787, -1.4519037008285522, -0.32855504751205444, 0.20941920578479767, 0.8947416543960571, 2.2118442058563232, 0.32194003462791443, 1.5302172899246216, 0.09805852174758911, -0.5841637253761292, -0.5296120047569275, 0.5205554962158203, -0.3100023567676544, -0.902836799621582, 1.159419059753418, -0.28691551089286804, -0.32742422819137573, 0.002647185465320945, -0.6030329465866089, 1.3667120933532715, 0.758793294429779, 1.2425602674484253, 0.9386645555496216, -1.1219009160995483, 0.46555086970329285, 1.1528950929641724, 0.36901015043258667, 0.5182139277458191, 1.4677194356918335, -0.30405429005622864, -1.0034778118133545, 0.7138639688491821, -0.5992038249969482, -0.18294377624988556, 0.5079386830329895, 1.2300636768341064, -0.22674450278282166, 0.2766645550727844, 1.2927664518356323, -1.937881350517273, 0.5739839673042297, 0.30119603872299194, 0.694850504398346, -0.8671945929527283, 1.3695403337478638, 0.6172223687171936, 0.9547821879386902, -1.1244210004806519, -2.133049726486206, 0.19869768619537354, -0.34054309129714966, -0.9404823780059814, -0.03712598979473114, -0.11139038950204849, -1.3554304838180542, 1.1725316047668457, 0.6768829822540283, -0.7116389274597168, 0.1787225753068924, 0.9855944514274597, 0.37969744205474854, -2.098059892654419, 2.191506862640381, 0.9156444668769836, 0.6783279180526733, -0.2586536109447479, 0.4599319398403168, -1.9053969383239746, -0.07363438606262207, 1.3530800342559814, 0.6794172525405884, -0.22362877428531647, 0.5760974287986755, 0.6097387075424194, 0.08840949088335037, 0.9103149175643921, -0.1682937890291214, 0.0978933572769165, 1.3964366912841797, -0.25904494524002075, 0.6491869688034058, 0.033998601138591766, 0.037029046565294266, -1.027213454246521, -0.7479353547096252, -0.5136979222297668, 1.3127961158752441, 1.2093744277954102, -0.22541746497154236, 0.016673456877470016, -0.9660618305206299, -0.5771124958992004, 0.12701450288295746, 0.8390762209892273, 1.1799004077911377, -1.0124411582946777, 0.548438549041748, -0.18264679610729218, -0.4205116629600525, 0.9327989220619202, -0.7210286855697632, -0.1126180961728096, 0.5321135520935059, 1.2209439277648926, 0.6455063223838806, -1.715004324913025, -1.1042704582214355, -0.11095357686281204, -0.6131658554077148, 0.38953155279159546, -0.9401296973228455, 0.6799008250236511, 1.8776382207870483, 1.4964390993118286, -0.49149808287620544, 0.9153850674629211, -0.19841937720775604, -0.7211349010467529, 1.4580854177474976, -0.7979438304901123, 1.4268285036087036, -2.8013594150543213, 1.1741771697998047, -1.4272934198379517, 0.21508769690990448, 0.2479179948568344, -0.05333526432514191, 0.33582940697669983, 2.097276210784912, -1.903844952583313, 0.39246970415115356, 0.9976869225502014, 0.4538830816745758, -1.532236099243164, 0.4485682249069214, 0.8013328313827515, -0.6663788557052612, -0.14043545722961426, -0.519774317741394, -1.8373007774353027, -0.014711803756654263, 0.05057000368833542, -0.035157978534698486, 0.5554024577140808, -2.0398075580596924, 0.013460882008075714, -1.1092886924743652, 0.7568973302841187, 0.725265383720398, 0.7197034358978271, -1.2772561311721802, 1.1695717573165894, 1.5785902738571167, 1.018794059753418, 0.2787056863307953, 0.6407328248023987, 0.18128694593906403, -0.864949643611908, 0.4185539186000824, -2.4788169860839844, 0.08647877722978592, -1.1629756689071655, 2.505915641784668, 0.030384093523025513, 2.0587310791015625, 0.7709172368049622, 0.026166077703237534, -0.13984425365924835, 1.1856722831726074, 0.05135173350572586, 0.4998827874660492, 1.8959752321243286, 1.8220943212509155, 0.13811416923999786, -1.312052845954895, -0.48686307668685913, -1.304756999015808, 1.2388967275619507, -2.21694278717041, 0.14019279181957245, -0.6469382643699646, 0.2097192406654358, -0.3573121130466461, -0.4715340733528137, 0.16896392405033112, -2.1940112113952637, -0.24583236873149872, 0.3016343414783478, 0.45417356491088867, 0.6638560891151428, -0.13042496144771576, 1.2019609212875366, -0.7707788944244385, 2.0332467555999756, 0.8246960639953613, -0.6799818277359009, 2.377056837081909, -0.6729630827903748, -1.2959740161895752, -1.3328698873519897, -1.7340869903564453, -0.4088226854801178, -1.817712426185608, -1.704181432723999, -0.7985876202583313, -0.7089002132415771, 0.1206786036491394, -0.7262177467346191, 0.045050282031297684, -0.20993643999099731, 0.20789575576782227, -1.4705766439437866, -1.366902470588684, -0.647509753704071, -1.6001890897750854, -0.6458104252815247, 0.3479492664337158, -0.8487639427185059, 1.0858064889907837, 0.7064741253852844, 1.1200029850006104, 1.0248079299926758, -0.05004424229264259, -0.7918140292167664, -0.4971078038215637, -1.511792540550232, -1.304663896560669, 0.2209513783454895, -0.8194406628608704, 0.8866060972213745, -0.604841947555542, 0.5844763517379761, -0.802798867225647, -0.5947663187980652, 0.7312706112861633, -0.11586279422044754, 0.13395461440086365, -0.4401686191558838, 0.13212257623672485, -0.5786786675453186, 1.0669646263122559, 0.4245772063732147, -0.39056557416915894, -0.30929481983184814, 1.576570749282837, -0.3823617100715637, 2.059469699859619, 0.22490264475345612, -0.9137850403785706, 0.7215625643730164, -0.240165576338768, 1.4639984369277954, 0.5500835180282593, 0.6870647072792053, -0.7558181285858154, -0.7111586332321167, -1.150813102722168, -0.9164360761642456, -0.7921853065490723, 0.9797840118408203, 0.6334612369537354, 0.3371466398239136, 0.2602454721927643, 0.1364416927099228, 1.1061687469482422, 0.6426714062690735, 0.3689618408679962, 0.8585730791091919, 1.0734446048736572, 0.21314097940921783, 1.544507622718811, -0.9823766350746155, 1.1340861320495605, 0.8288238048553467, 0.30482250452041626, 0.23139536380767822, -1.1926292181015015, 0.4228004217147827, -0.17637556791305542, 1.4894757270812988, -0.31076598167419434, 1.7759817838668823, 1.080370545387268, 0.0008967131143435836, -0.28343266248703003, -0.347379207611084, -0.1844540238380432, -1.7799906730651855, 0.6456313729286194, -0.6260249614715576, -1.2390459775924683, -1.557962417602539, 0.6211775541305542, 0.06915098428726196, 0.8168987035751343, -0.10519180446863174, 1.129802942276001, -0.9206746220588684, 0.5780007839202881, 0.9311870336532593, 0.10992804914712906, 1.0841779708862305, -1.0799685716629028, -0.006010120734572411, 0.2170276641845703, 0.2309744954109192, -1.3700551986694336, 1.02243173122406, -0.9990359544754028, -0.303476482629776, -1.8074657917022705, 1.5401418209075928, 1.2450956106185913, -0.7365502715110779, -0.4738248884677887, -0.021588001400232315, 1.0563095808029175, 1.3173211812973022, 0.49412453174591064, -0.8936166763305664, -0.9613773822784424, 0.6893664002418518, 1.8426705598831177, -2.349547863006592, -0.2130039483308792, 2.001227617263794, 1.8599869012832642, -0.5140812397003174, 0.034498024731874466, -0.6012898683547974, -2.4724724292755127, -0.36209428310394287, -0.08945734053850174, -1.5263515710830688, 1.5124891996383667, 0.6756187677383423, -0.15200264751911163, 1.2963991165161133, 0.7120950818061829, -0.006494136527180672, -0.7835356593132019, 1.0689893960952759, 0.21605746448040009, -0.20070919394493103, 0.5684590935707092, 0.9306426644325256, 1.1770600080490112, -0.9378352761268616, 0.46868154406547546, 0.8059859275817871, -1.3496508598327637, 0.37632113695144653, -1.5054417848587036, -0.23758159577846527, -0.8521984815597534, -0.880675733089447, 0.5529263615608215, -0.5355082750320435, 0.3348311185836792, -0.40253472328186035, -0.8942349553108215, 0.47797566652297974, 0.3173271119594574, 1.557532787322998, 2.170698404312134, 1.3813071250915527, -0.42893749475479126, 0.4191421568393707, 0.7552621364593506, 0.3366452157497406, -0.5245067477226257, -0.7717613577842712, -0.5071592926979065, -0.8085202574729919, -0.763411819934845, -0.2657100558280945, -0.14722132682800293, -0.5972578525543213, 1.0321109294891357, -0.44595587253570557, 1.053429126739502, -0.5780790448188782, 0.4458574652671814, 0.9170467853546143, -0.44985947012901306, -0.9439972639083862, 0.32761794328689575, -0.6936434507369995, -0.33153796195983887, -0.6335088610649109, 0.9285176396369934, -0.43756094574928284, -0.1748885065317154, -0.9935327768325806, -1.2875384092330933, 1.0052577257156372, 0.7717658877372742, -0.25101202726364136, -0.11021585017442703, -0.3307105004787445, 0.22084613144397736, -0.3611031770706177, 0.5007266998291016, -0.006888343021273613, 2.0592994689941406, 0.6149722933769226, -0.27060434222221375, -2.4839460849761963, 0.45683929324150085, 1.348961353302002, -0.22986187040805817, -0.16257742047309875, -2.057368278503418, -0.7053006887435913, -0.6718482971191406, -0.944512128829956, -1.089766502380371, -0.16190087795257568, -1.1799041032791138, -2.1725237369537354, 0.5063256621360779, -0.5415717363357544, 0.19043028354644775, -1.2905384302139282, 1.5931733846664429, 0.33934131264686584, 0.48330947756767273, 1.0151463747024536, 0.5178120136260986, -1.085447907447815, 2.1970620155334473, 0.02125009521842003, 1.6707334518432617, 0.4804393947124481, 0.5762558579444885, 0.6012437343597412, -0.7467536926269531, 0.0870044156908989, 0.03272126242518425, 0.08831099420785904, 0.3366429805755615, -0.3432671129703522, 0.6345070004463196, -1.4873154163360596, 0.15286357700824738, -2.8012032508850098, -0.28420108556747437, 0.3611263036727905, 1.64395010471344, 0.5371205806732178, -1.4077701568603516, 0.3194204866886139, 0.6048350930213928, -0.5424823760986328, -0.028124021366238594, -0.6486515998840332, 0.8654628396034241, -0.10927138477563858, -1.820536732673645, -0.8763333559036255, -2.226351499557495, 0.34244048595428467, 0.22983123362064362, -2.2125940322875977, -0.3015783429145813, -1.0967165231704712, 1.2532106637954712, -0.5936400294303894, -1.1081299781799316, 0.3532896637916565, 1.4228453636169434, -1.9944056272506714, -0.5335618257522583, 0.21091996133327484, 0.0765879899263382, -2.011021852493286, -0.5631119012832642, -0.6912747621536255, 0.3833218812942505, 1.1743263006210327, 0.1375938355922699, 0.6246901154518127, -1.2235300540924072, 0.15057535469532013, -0.9605415463447571, -0.5610741972923279, 1.0542492866516113, 0.20701651275157928, 1.0295342206954956, -0.6719393730163574, 0.9061434268951416, -1.021661400794983, 0.14285801351070404, 1.5404809713363647, 0.786620557308197, -0.6443038582801819, -0.45221465826034546, -0.6329972743988037, 1.6149893999099731, -0.5131277441978455, 0.1428852379322052, 0.30890804529190063, -1.3917818069458008, -0.5878518223762512, -0.8174559473991394, -0.041842807084321976, 0.08930016309022903, -0.11775913834571838, 0.04294336959719658, 0.6930646896362305, -0.06620031595230103, -0.39451348781585693, 1.6295946836471558, -1.7705553770065308, 0.556886613368988, 0.4633888304233551, -0.21921010315418243, -0.32339078187942505, 0.4338841736316681, 1.1687754392623901, -0.8036704659461975, 0.8360201716423035, 1.9291232824325562, 0.3878448009490967, -0.5435743927955627, -0.5349939465522766, -0.37636232376098633, -0.7596092820167542, 0.22095011174678802, 0.5388956665992737, -0.2246476262807846, 0.2412465512752533, 0.6998096108436584, -0.6433852314949036, -0.6566966772079468, -0.8012146353721619, 1.018149971961975, -2.2953081130981445, -0.388208270072937, -0.2802899181842804, -0.5273818969726562, 2.0747435092926025, -1.1250540018081665, 0.3302912414073944, -2.8475286960601807, 0.71657794713974, -0.24381810426712036, -0.4312853217124939, 0.21979685127735138, 1.3412257432937622, -0.7952980399131775, -0.8882381319999695, -3.035825252532959, 0.5210602879524231, -0.026161957532167435, 0.16334281861782074, 0.24671489000320435, 1.1918057203292847, -0.3952985107898712, -1.1408520936965942, -0.2636549472808838, -0.3436329662799835, 0.15289314091205597, 0.7208375334739685, 0.1988915205001831, -1.3486465215682983, 0.11600203812122345, -0.5870984792709351, 0.5095099806785583, -1.2789210081100464, -0.9864313006401062, 1.1264549493789673, 1.27707040309906, 0.6075674891471863, -0.3417234420776367, 1.18025803565979, -0.45713749527931213, -0.005044324789196253, -0.5865454077720642, 1.1377002000808716, 0.1319272816181183, -1.3952727317810059], [-1.3473964929580688, -2.0834786891937256, 2.628857374191284, 0.3523271679878235, 2.1360764503479004, 1.081969976425171, 0.9663234949111938, -1.0311846733093262, -0.973057746887207, -0.755199134349823, 0.034397464245557785, 1.7287298440933228, 1.2298585176467896, 0.4338718354701996, -1.0398396253585815, -0.10221795737743378, 0.7402129769325256, 0.4280712902545929, 0.812103271484375, 0.27711302042007446, 0.7259777188301086, 0.10110155493021011, 1.716407299041748, 0.44271233677864075, -0.48509031534194946, -0.24315638840198517, 2.4114835262298584, 1.4953973293304443, -0.5630574226379395, 1.166333556175232, 1.4307008981704712, 1.6218552589416504, 0.2493685483932495, 1.9063447713851929, 0.38236990571022034, 0.4121047556400299, 0.3149343729019165, -1.0795031785964966, 0.480236291885376, -1.4333040714263916, -1.2195533514022827, -0.6533948183059692, -1.2881594896316528, 0.17871597409248352, 0.6154017448425293, -0.06939946115016937, 0.6670984029769897, 2.341742753982544, -1.5001907348632812, 1.3192863464355469, 0.23929251730442047, -0.43878814578056335, 0.13770519196987152, 0.3004242777824402, -0.13458764553070068, -1.2668849229812622, -0.2963871955871582, -0.5985866189002991, 0.2828003466129303, -0.20786523818969727, 0.9991339445114136, -0.5590419769287109, -0.3868984878063202, 0.4046933054924011, 1.6314646005630493, 0.9028745889663696, -1.5955867767333984, -2.607811212539673, 0.19638240337371826, -0.13843536376953125, 0.02203604020178318, 1.5702803134918213, -1.078270673751831, 0.5132057666778564, 1.298163652420044, -0.09783986955881119, 1.1681218147277832, 1.3655434846878052, 0.5541487336158752, -1.1829242706298828, 0.7539631724357605, 0.6332292556762695, 1.0418603420257568, 0.1498020738363266, -0.6184182167053223, -0.5184942483901978, 1.364960789680481, 0.8143585920333862, -0.06819619983434677, 0.42420950531959534, 1.638474702835083, 0.6590157747268677, 0.32151180505752563, 0.5613381266593933, -0.7123252153396606, -1.1036045551300049, 0.6769773364067078, 0.6875385046005249, -0.11071785539388657, -1.209520697593689, -0.46792322397232056, 1.6558114290237427, 1.499521017074585, -1.4404199123382568, -0.3097681403160095, 0.5161193609237671, 0.03925883769989014, -0.3031502962112427, 1.2507306337356567, 0.9948975443840027, -0.32161882519721985, 0.7471251487731934, -0.24490652978420258, -0.024127712473273277, 0.45083707571029663, 1.1733007431030273, 0.08981786668300629, 0.5919855237007141, -0.5649600028991699, 0.29406285285949707, -0.8909570574760437, 0.6043742895126343, 0.30931535363197327, 0.5245682597160339, -0.5258165001869202, -0.6728941202163696, 0.652394711971283, 0.501290500164032, -0.6669867634773254, 0.8653343915939331, 0.7972362637519836, 3.426328659057617, -1.0927006006240845, 0.01666238158941269, -0.9173664450645447, 0.782265841960907, 1.080119252204895, 1.9312114715576172, 1.708891749382019, 1.9801126718521118, 2.0944130420684814, 0.26216965913772583, -0.0040609887801110744, -0.9459669589996338, 0.001643423456698656, -0.8138779401779175, -1.6204901933670044, 2.1437926292419434, -0.09954833984375, -1.0682604312896729, 0.8657593131065369, 0.1394316703081131, -0.3463682234287262, -1.5047634840011597, -0.40750381350517273, -0.17184282839298248, 0.20552749931812286, 1.496716856956482, -1.2891921997070312, -0.529400110244751, 0.006994565948843956, -0.3646286427974701, -1.3298081159591675, 0.8066650629043579, 0.17555934190750122, 0.5813363194465637, 0.4298737347126007, -0.4939238429069519, 0.4736328423023224, -1.5758918523788452, 0.4851081073284149, -1.294492244720459, 0.37352728843688965, 0.009829756803810596, 1.035422921180725, -0.396750807762146, -1.2988312244415283, 0.30059683322906494, 0.6775310039520264, 0.7877957820892334, 1.1328045129776, -1.5301629304885864, -1.1127480268478394, -1.7112369537353516, 0.651096522808075, 1.1869393587112427, -1.17731773853302, -0.4595070779323578, 1.4281811714172363, 1.9288437366485596, -0.17780908942222595, 1.0203486680984497, -0.906821608543396, 0.5426275730133057, -0.07730882614850998, 0.36188241839408875, -0.2856270968914032, -0.2458549588918686, -0.8159599304199219, -0.16341856122016907, 0.20440328121185303, 1.900686502456665, 1.9671109914779663, 0.8425265550613403, -1.027796983718872, 0.2093096673488617, 0.8592066168785095, 0.39381080865859985, -0.9803980588912964, 1.4173643589019775, 0.9798718094825745, 1.0889476537704468, 1.8246726989746094, 1.24430513381958, -0.08262570202350616, 0.060535650700330734, 0.2000521719455719, 0.56889808177948, -0.1452031135559082, -1.1491845846176147, 0.6879785656929016, 0.5039786696434021, 1.0251762866973877, -0.5683094263076782, -1.4616413116455078, 0.020654357969760895, 1.5687617063522339, -0.05655380338430405, 1.26047682762146, -0.8621883392333984, -0.8022573590278625, -0.6924216151237488, -0.3313848078250885, -1.3479342460632324, 1.921543836593628, 0.4621213972568512, 0.4123806059360504, 1.081387996673584, 0.17832247912883759, 0.029965639114379883, 0.7052105069160461, 0.940861165523529, 0.16254201531410217, 0.4346869885921478, -0.25042372941970825, 1.3783107995986938, -1.4232043027877808, 0.5163042545318604, -0.12051025032997131, -0.26005885004997253, 0.37780967354774475, -1.1472851037979126, -1.0101747512817383, 0.896159291267395, -1.1926246881484985, -0.45110848546028137, 0.10437352955341339, -1.007802128791809, -1.0362406969070435, -0.84114009141922, -1.9631600379943848, 0.822921633720398, 1.189319133758545, 0.6480823755264282, -1.7564066648483276, -1.7580814361572266, -0.23670227825641632, 1.0139111280441284, -0.8292453289031982, 1.3072506189346313, 0.3700551986694336, 0.5776299834251404, -0.6175264120101929, -0.6459628939628601, -1.885949969291687, -0.9648903608322144, -1.1445786952972412, -0.5123223066329956, -0.3821071982383728, -1.8985587358474731, 0.15896981954574585, 0.3238391578197479, -0.3191986382007599, -0.1685645580291748, -1.5584356784820557, -1.6455966234207153, 0.4799344837665558, 0.45603135228157043, -1.0973436832427979, -0.24749518930912018, 0.41625499725341797, -0.45151403546333313, -1.4066317081451416, 0.5711981654167175, -0.31925809383392334, -0.2619819641113281, -1.4543980360031128, 1.3734686374664307, 0.6835036277770996, 0.6273784041404724, -0.7802729606628418, -0.48318350315093994, 0.4703586995601654, 0.30904707312583923, -0.6366066932678223, 1.0967825651168823, 0.14851249754428864, 0.08596453070640564, 0.8028615117073059, -1.1806827783584595, 0.47945845127105713, 1.6544811725616455, 0.6579059958457947, 0.5835670232772827, -1.2334309816360474, 0.26729539036750793, -1.3126530647277832, -0.3877630829811096, -0.17735157907009125, -0.09469650685787201, 0.7636796832084656, -0.6444116234779358, 2.5850329399108887, -0.5510298013687134, 0.5735707879066467, -0.5447599291801453, 0.2522240877151489, -0.28653231263160706, -1.7135177850723267, 1.6039875745773315, 0.3149370551109314, -0.7529662847518921, -1.1502928733825684, 0.5507010817527771, -1.5087841749191284, -0.48582834005355835, 2.363579273223877, -0.9674307703971863, -0.5428339242935181, 1.9738630056381226, 0.8050571084022522, -1.519006371498108, -0.051168523728847504, 0.04523364081978798, -1.0654709339141846, -0.806904137134552, -0.14372694492340088, 0.3204215168952942, -1.6311988830566406, -0.2677103281021118, -0.3607015311717987, 1.305163025856018, 0.11764692515134811, -0.3031853139400482, -2.1436715126037598, -0.4888194501399994, -1.0838212966918945, 0.06603377312421799, 0.9216030240058899, 0.2903239130973816, 0.19328761100769043, -0.5309613943099976, -1.490767240524292, -0.8079485893249512, -0.17301756143569946, -0.1683448851108551, 1.2637425661087036, 0.5536233186721802, -0.34716957807540894, 0.16447439789772034, -0.16681025922298431, 0.5934805870056152, 0.15506160259246826, -1.158948540687561, 0.5611295104026794, -0.12111883610486984, 0.45303401350975037, 1.76754891872406, 0.22840924561023712, -1.183398723602295, -0.03240928053855896, 0.6297153234481812, 1.210011601448059, 1.060166358947754, 0.6434550285339355, -0.7936754822731018, 0.2965107858181, -1.512749433517456, 0.24096131324768066, 0.34922683238983154, -0.6595319509506226, 0.7352287769317627, 0.5318271517753601, 0.10272125899791718, -0.5874324440956116, 1.3341866731643677, 1.061608076095581, 0.561074435710907, 0.44160687923431396, 1.4534659385681152, 0.8737039566040039, -0.7840419411659241, 1.550254464149475, 0.717165470123291, -0.05875921994447708, -1.931738018989563, 0.5763707756996155, 0.41890785098075867, -1.2419991493225098, 0.8065792322158813, 0.5458745360374451, 0.7924643754959106, 1.0487513542175293, 1.0476713180541992, 0.3771759867668152, 1.4888408184051514, -0.25970184803009033, 1.3623520135879517, -1.3925122022628784, 0.7719507813453674, 0.6749066710472107, 0.04894746467471123, -0.36573153734207153, 0.37895530462265015, -0.2610792815685272, 0.4768679738044739, 1.3785769939422607, 0.3166627287864685, 0.7174371480941772, 0.3645477890968323, 0.031154129654169083, -0.7277699708938599, 0.8678288459777832, -1.6262398958206177, -0.9428739547729492, -0.23342746496200562, -0.16411840915679932, 0.15682236850261688, -1.7777949571609497, 2.2262885570526123, 0.32341769337654114, 1.4760688543319702, 0.14223769307136536, -1.3739136457443237, 1.0516804456710815, 1.2883474826812744, -1.507381796836853, -0.641168475151062, 1.4527204036712646, -0.32178470492362976, 1.0842097997665405, 0.34862977266311646, 0.042131755501031876, -1.1045812368392944, 0.33634454011917114, 1.0870354175567627, 1.407584309577942, 1.6420584917068481, 0.2253149300813675, 0.4847528040409088, 0.08347770571708679, -0.16415826976299286, 1.301716685295105, 0.5357749462127686, 0.30509456992149353, -1.6446489095687866, 0.353920042514801, -0.9829939007759094, 0.520827054977417, -0.38699212670326233, -0.14797089993953705, -0.14092400670051575, 0.6456077098846436, 0.6289829015731812, -0.2606298327445984, -2.2697055339813232, 2.4190988540649414, 0.7606155276298523, 0.30614230036735535, 0.9499463438987732, -0.939507007598877, -1.0705012083053589, 0.7254308462142944, -0.5522241592407227, 1.3178976774215698, -1.5264496803283691, -0.4472722113132477, 0.3053570091724396, -0.40167343616485596, -1.474181056022644, 0.39645639061927795, 0.6672547459602356, 0.4292774498462677, 0.7164644598960876, -1.8465328216552734, -0.0913325697183609, -1.816987156867981, -0.5533117055892944, -0.41603097319602966, -0.6622929573059082, -0.6215270757675171, 0.4871336817741394, 1.6293396949768066, -0.17834216356277466, 1.2770284414291382, -1.107523798942566, -0.9293273091316223, -0.5492051839828491, 1.241878628730774, -0.3368786871433258, 0.2589718997478485, -1.8244136571884155, -0.7015266418457031, -0.9136477112770081, 1.4517334699630737, 0.6326656937599182, -0.9058164358139038, 0.7941274046897888, 0.6764453053474426, -0.9123349785804749, 0.13924118876457214, 1.458986520767212, -1.0670713186264038, 0.5487496256828308, -1.2717249393463135, -0.6539446711540222, 2.7248802185058594, 0.16834819316864014, 0.8165349364280701, -0.8765493631362915, 1.0715004205703735, -0.16427099704742432, 3.0555272102355957, -1.7135595083236694, 1.1301926374435425, -0.6926103234291077, 1.4610148668289185, -0.7539833188056946, 1.713281512260437, 0.03144427388906479, 0.49772682785987854, 1.28656005859375, 3.3103384971618652, 1.4502439498901367, -0.9266616106033325, 0.01773860864341259, -0.35686618089675903, -0.22504687309265137, 0.8522019386291504, -0.7426448464393616, -0.5706374049186707, 1.106444001197815, -1.0706740617752075, -0.5424023866653442, 1.033255696296692, -0.20221750438213348, 0.2332424372434616, 0.20332281291484833, 0.28767043352127075, -1.4480186700820923, -1.05100417137146, 0.018462495878338814, -0.5547774434089661, -0.23936398327350616, 0.8694638609886169, -0.132533460855484, 0.08670419454574585, 0.7580042481422424, -0.5739043354988098, 0.3633102476596832, 0.8193536400794983, -1.634684681892395, -0.31978222727775574, 0.09636670351028442, 0.9837158918380737, -0.8035993576049805, 0.8137202858924866, 0.6031154990196228, -0.33491969108581543, -0.6871243715286255, 1.2910749912261963, 0.031177163124084473, -0.3957606554031372, 0.043037816882133484, 0.020555639639496803, -0.08406537771224976, -0.6723120808601379, -1.2178853750228882, -1.4181673526763916, -0.4740371108055115, -0.39798834919929504, -0.9609161615371704, -0.7470163702964783, 1.7352415323257446, -0.36563241481781006, -0.7879770398139954, 0.14477208256721497, 0.0007570340530946851, 0.7635236978530884, 0.6448713541030884, -0.37380510568618774, -0.09685730934143066, 1.5168043375015259, 0.9723625779151917, 0.6627514958381653, -0.30758264660835266, -0.8439632654190063, -1.277742624282837, -0.40398478507995605, 0.380687415599823, -0.10998373478651047, 0.8885554671287537, -1.0758823156356812, 0.8659830689430237, -2.0658862590789795, 0.3735013008117676, -1.2090200185775757, -1.4000191688537598, -1.1506829261779785, -0.7475268244743347, -0.802676796913147, 0.9336307644844055, -0.19641149044036865, -0.30052775144577026, -1.296662449836731, -1.0275139808654785, -0.0789337307214737, 0.9061943292617798, 1.267072319984436, -0.9741021394729614, 0.4529263973236084, -0.4789743721485138, -0.6626102924346924, -0.5876013040542603, -0.3713173568248749, -0.13420765101909637, 0.014378773979842663, -0.7415177226066589, 0.6297420859336853, -0.40502190589904785, -0.44194120168685913, 0.8389168381690979, -1.3512665033340454, 0.5309325456619263, -1.3698447942733765, -1.5647014379501343, -2.912997007369995, 0.1740204244852066, -1.1348156929016113, -0.9495303630828857, -0.452968567609787, 1.0013552904129028, 0.725607693195343, 0.5439586639404297, -1.1886074542999268, -1.2649412155151367, 0.1394665688276291, 0.9882825016975403, -0.31599119305610657, 1.1101752519607544, -0.6814144849777222, 0.9370728135108948, 1.1005295515060425, 0.0031530559062957764, -0.5385741591453552, 0.6870185732841492, 0.4236934185028076, -0.43582436442375183, 0.6602838039398193, 0.21767868101596832, 0.14437933266162872, 0.9234289526939392, -0.3915170133113861, -1.315293788909912, -1.1146384477615356, 2.673464775085449, 0.494639128446579, 0.24989046156406403, -1.0896281003952026, 1.9743496179580688, 1.2421921491622925, 0.7556931376457214, 2.3737099170684814, -0.08997903764247894, 0.8445229530334473, 0.5441938042640686, -1.3853225708007812, -1.4877219200134277, 1.210215449333191, 0.0777059942483902, -0.3781532645225525, -0.5087572932243347, 0.9886146187782288, -0.0980459675192833, -0.9991267323493958, -2.2629799842834473, -0.4428860545158386, 0.5401289463043213, 2.290329694747925, 2.276033878326416, -0.9000676274299622, 0.4355943202972412, -1.2736810445785522, 1.30867600440979, -0.10187279433012009, 1.117600440979004, 0.0005674958229064941, 0.6151249408721924, -1.3740673065185547, 1.6104689836502075, -0.3171888589859009, 0.21457606554031372, 0.08147002011537552, -0.7912587523460388, -0.8033556938171387, 0.9452691674232483, 0.14566214382648468, -1.6821911334991455, -0.06043441593647003, -0.5844369530677795, 1.6818610429763794, -0.49396374821662903, 0.0917138010263443, 0.7331690192222595, -1.118109107017517, 0.5747159719467163, 2.738135576248169, 0.910090982913971, 0.4366861879825592, -1.6097453832626343, 0.07279138267040253, -0.9473259449005127, -0.35162538290023804, 0.007860197685658932, -0.5727975964546204, 0.7447547912597656, -0.5633535385131836, -0.35045334696769714, -1.6201848983764648, -0.061707619577646255, -1.4422112703323364, -0.8968843817710876, 0.8097956776618958, -0.3374633193016052, 0.15106026828289032, 2.248908281326294, 1.1894861459732056, 0.19684460759162903, 0.5949456691741943, 0.19960740208625793, -1.4057420492172241, 0.04044065997004509, -1.2227221727371216, 0.2857495844364166, 0.38981571793556213, -0.44814932346343994, -0.31723693013191223, 1.16195547580719, -0.9607227444648743, 1.0194554328918457, 1.357599139213562, -0.4842081069946289, -1.2586716413497925, 0.31548216938972473, 0.0808950737118721, 0.25582849979400635, 0.13889814913272858, -0.216436967253685, -1.1472779512405396, 0.4540765583515167, -0.9956634640693665, 1.0837849378585815, 0.7910308241844177, 0.502911388874054, -0.7881028056144714, -1.0247306823730469, 0.5840346217155457, 0.8247473239898682, -0.003014330519363284, 0.2182437926530838, 0.9868383407592773, 0.3687038719654083, -0.05816765874624252, 1.6512428522109985, 0.8893848657608032, 1.7153418064117432, -0.35119614005088806, -0.12142566591501236, 0.32299330830574036, -2.1823670864105225, 0.8537711501121521, -0.1079404354095459, -0.437162309885025, 0.37672001123428345, 0.14388835430145264, -0.4912700951099396, 0.1507590413093567, 0.809928834438324, 0.4569263756275177, 0.09243745356798172, -0.93971848487854, -0.06718246638774872, 0.892314076423645, -0.9558863639831543, 0.7838636636734009, -0.2654212415218353, 3.2184011936187744, 0.32201921939849854, 1.171693205833435, 1.0836751461029053, 0.5964949727058411, -0.8898001313209534, -2.3267531394958496, -0.8408300876617432, -1.5134204626083374, -0.8696117401123047, 2.1402506828308105, -0.9008852243423462, -1.1994287967681885, -1.1804574728012085, -0.47695478796958923, -1.2015652656555176, -0.1791898012161255, -0.6046770811080933, 1.3052828311920166, 0.5836916565895081, 0.6098827719688416, -0.1419249325990677, 0.6741758584976196, 0.1345570683479309, 0.18847766518592834, 0.033022526651620865, 0.1868879646062851, 0.4127216637134552, 1.254712700843811, -0.21786515414714813, -0.749437689781189, -0.9823881983757019, 0.8593045473098755, 0.8397036194801331, 1.6776880025863647, 0.566480815410614, -1.6979960203170776, -0.07678859680891037, 0.3091254234313965, -1.1802809238433838, 1.1392062902450562, -1.125245451927185, -0.39387550950050354, 2.21236252784729, -0.21887478232383728, -0.6135793328285217, -1.3646827936172485, 0.17880038917064667, -0.8614226579666138, -0.12382493168115616, -0.13146676123142242, -0.556355357170105, -0.19382527470588684, -0.38972941040992737, -0.9235929846763611, 1.1099421977996826, 1.4779250621795654, -0.4925774931907654, -0.21817736327648163, 0.1083562895655632, 1.1618552207946777, -2.2852842807769775, 0.03372872620820999, -0.543256938457489, -1.2140051126480103, 0.39703479409217834, -2.1218178272247314, 0.7836831212043762, 1.3295269012451172, 1.0895721912384033, -0.7222389578819275, -0.6240167617797852, 0.8968797326087952, -0.5981586575508118, 0.009000004269182682, 0.5314910411834717, -0.0480831004679203, 0.6504939198493958, 1.8328708410263062, 1.4023411273956299, 1.3145084381103516, -0.017664596438407898, 0.6190974116325378, 1.4941349029541016, -1.8678607940673828, -0.383327841758728, -0.13717485964298248, 0.2502785623073578, -1.0133079290390015, 0.5800153613090515, 0.2835107147693634, 0.6920108199119568, -0.7244510650634766, -0.03715309873223305, -1.4779082536697388, 0.49682316184043884, 0.9486607909202576, 0.09780482947826385, 0.36944034695625305, 0.1215374544262886, 1.1959558725357056, -0.3199094533920288, 0.3096635937690735, -0.8448987007141113, -1.2250014543533325, -1.1150227785110474, 2.0601794719696045, -0.4143611192703247, 0.8689066767692566, 0.1418810933828354, -0.5651800036430359, 2.180835247039795, -0.7735010385513306, 0.22178727388381958, 0.1569085717201233, 0.404051810503006, -0.34013715386390686, -0.2695476710796356, 0.7798428535461426, -0.12218687683343887, -1.6200392246246338, -0.2596881091594696, -0.8352294564247131, -0.7254136204719543, 0.06575644016265869, 0.5793285369873047, 1.8515448570251465, -1.0911896228790283, -0.8082037568092346, -0.2836684584617615, -0.39706307649612427, -0.8333429098129272, 0.8254638910293579, -1.1292469501495361, -0.7164276242256165, 0.17114052176475525, -0.8615919947624207, -0.7933480739593506, -0.9956249594688416, -0.49106690287590027, -0.023222025483846664, -1.6772818565368652, -0.18182246387004852, 0.05094931647181511, 0.5673580169677734, 1.5596153736114502, 0.5700032711029053, -1.2207636833190918, -0.9885040521621704, -0.49534207582473755, 0.5111076235771179, 0.03315316513180733, -0.496581107378006, 0.19004486501216888, -1.4323002099990845, 0.05149988457560539, 0.4673534333705902, 1.4449645280838013, 0.05394815281033516, 0.13940474390983582, 0.5546044111251831, -0.18108011782169342, 0.22615531086921692, -1.0962481498718262, 0.5144367814064026, -0.9756755828857422, -0.6406387090682983, -0.5406690239906311, -0.36540645360946655, 1.0730093717575073, 0.41442790627479553, -0.8616722226142883, -1.7709652185440063, -0.5160067677497864, -1.1874494552612305, -1.1238296031951904, -0.7551361918449402, -0.7863906621932983, 1.6499897241592407, 0.42862534523010254, 0.12506619095802307, 1.0444446802139282, 0.35132896900177, -0.47769442200660706, 0.31335386633872986, 1.4516708850860596, -0.6166322231292725, 0.3861604630947113, -1.7292494773864746, 1.2874891757965088, -1.2245383262634277, 1.106479287147522, 2.4342737197875977, -0.667129397392273], [-0.09997601062059402, -0.5103431940078735, -1.6295164823532104, -0.5806277394294739, -0.4843493402004242, -0.49108773469924927, -0.22598712146282196, -0.7824872136116028, 0.45168638229370117, -0.132857546210289, -1.651698112487793, -0.13157491385936737, 2.2363839149475098, -2.420391798019409, -0.059086937457323074, -0.2095889300107956, -0.48057395219802856, -0.5172240138053894, -1.2780463695526123, 1.4913216829299927, 0.047834936529397964, 1.1048028469085693, 0.011493957601487637, -0.6936022639274597, 1.272851824760437, -0.1233324408531189, -0.6249961853027344, 1.5596871376037598, -0.837821900844574, 0.34303972125053406, -0.3620871305465698, -0.4476673901081085, -0.5827251076698303, -0.2532027065753937, 0.6099963784217834, 0.6626138091087341, 0.48949292302131653, -0.3457758128643036, 1.2170051336288452, 0.9311913251876831, -0.4291297495365143, -0.8102499842643738, 0.008281216956675053, -0.25674766302108765, 1.145845651626587, -0.05633534863591194, 0.8913483619689941, 1.5070663690567017, 1.1869258880615234, 1.1044121980667114, -1.0996413230895996, 1.2643851041793823, -0.07916492968797684, -1.918877124786377, -0.6385626792907715, 1.0816386938095093, 0.3060935437679291, -1.3670629262924194, -0.6022304892539978, -0.26205551624298096, 0.741113007068634, 0.5772491097450256, 1.6192283630371094, 0.27202993631362915, 0.20571570098400116, 0.15236996114253998, 0.5819262862205505, -0.06498079746961594, -0.3081423342227936, -0.21831044554710388, -1.3475760221481323, -0.32366839051246643, 1.7103663682937622, -0.4334259331226349, 0.7609174251556396, -1.4859319925308228, -0.38593947887420654, -0.8170901536941528, 0.18760764598846436, -0.0794464498758316, -0.8747221827507019, 0.5891240239143372, -0.9696511626243591, 0.8335559368133545, -0.5421091914176941, 1.6812599897384644, -0.5637431144714355, 0.07842518389225006, 0.6521306037902832, 0.18351885676383972, 0.5743730664253235, 1.1981793642044067, 1.746469497680664, 0.7510237097740173, -1.6974340677261353, -2.059514284133911, 1.386638879776001, 0.4743711054325104, -0.37265291810035706, 0.3398124873638153, 0.034298256039619446, 0.6000297665596008, 0.22659865021705627, -0.8234406113624573, 1.275215744972229, 1.9500482082366943, 1.0770069360733032, 0.7567887306213379, 1.0371149778366089, -0.027758143842220306, -0.3042450249195099, -0.7099637985229492, -2.166473150253296, -0.8291230797767639, -0.9954708814620972, 0.24534574151039124, -0.19521552324295044, -0.03670987859368324, 0.17161445319652557, 0.8321357369422913, 0.4822927415370941, -0.577197253704071, 0.4905835688114166, -0.3973044753074646, -0.22499780356884003, 1.1173232793807983, -1.3012449741363525, -0.7977633476257324, -0.4229516088962555, 1.815105676651001, 0.38015392422676086, -0.002981110941618681, 0.6407151222229004, 1.3812707662582397, -1.079227328300476, 0.7291883826255798, -2.075470209121704, 0.833373486995697, -0.6744961738586426, 0.41757556796073914, 0.4053976535797119, 1.672436237335205, 0.3122568130493164, 0.21356289088726044, 0.9785968661308289, 0.004208265338093042, -1.0099101066589355, 0.24914011359214783, -0.0025586446281522512, 0.28102198243141174, -0.9458786845207214, 0.823657214641571, -0.0798935815691948, -0.7433118224143982, -2.6301565170288086, 0.8935410976409912, -0.342753529548645, -1.0220260620117188, 0.8079034686088562, -0.7943665981292725, -0.19363081455230713, 1.4026516675949097, 0.4492594003677368, 0.5199502110481262, -0.5433613657951355, 1.3982938528060913, -1.4031081199645996, -0.7686882615089417, -1.2039567232131958, -0.2834111750125885, 0.20768670737743378, 1.0415524244308472, 0.06629560887813568, -0.9364404082298279, -1.28005850315094, 1.6937824487686157, 1.1779868602752686, -0.04401421174407005, 0.9277153611183167, -1.4520741701126099, -0.5366726517677307, -0.057118769735097885, -0.39542120695114136, 0.3142828345298767, -0.254316121339798, 0.11036718636751175, -0.2540977895259857, -1.9467048645019531, 0.464690238237381, 0.49607324600219727, -1.3325024843215942, 0.9048122763633728, 1.0670701265335083, -1.0929621458053589, -0.4176541864871979, 0.44742727279663086, 0.40468287467956543, -0.7711054682731628, 0.27538710832595825, -0.06779338419437408, -1.0420172214508057, 1.2093874216079712, 0.6038228869438171, -0.24548234045505524, 0.5310737490653992, -1.22738516330719, -0.8846093416213989, 0.9019731283187866, 0.7496806979179382, 0.7908835411071777, -0.07706914842128754, -0.26223304867744446, -0.7016972899436951, -0.6049840450286865, -0.6693873405456543, 0.41491612792015076, 0.05142224580049515, 0.6063671112060547, 0.11407317221164703, 0.11693645268678665, -0.06332952529191971, 0.13430853188037872, -2.7069613933563232, 1.1422744989395142, 0.9425661563873291, 0.4219982922077179, -0.21094661951065063, 0.4652606248855591, -1.0324279069900513, 0.3958169221878052, -0.874613881111145, 0.35885265469551086, -0.9904769062995911, 1.3410019874572754, 0.13281889259815216, -1.9204477071762085, -0.7945418357849121, -1.1505650281906128, -0.5636228322982788, 1.498095989227295, -0.4657162129878998, 0.4343625605106354, -0.7990301251411438, -0.4369508624076843, 0.03913464397192001, -0.5468941330909729, -2.2334134578704834, 0.22498244047164917, -1.3591419458389282, -0.6897994875907898, 1.1658308506011963, 1.447884202003479, 0.6117265224456787, -1.2340130805969238, 0.12916694581508636, 0.6604732871055603, -0.19832873344421387, -0.8928367495536804, 1.964225172996521, 0.06739670783281326, -1.9649686813354492, 0.934087336063385, -0.02226828597486019, -0.9215539693832397, -0.1285475343465805, 1.6734081506729126, -0.6272115111351013, 0.4410807192325592, -2.34814715385437, 0.8588569760322571, -1.1436208486557007, -0.05082031339406967, 1.546474814414978, 0.45808809995651245, 0.8430805802345276, 0.12201941013336182, 1.1623408794403076, 0.4166322648525238, -2.686868190765381, -0.41410595178604126, 0.8430135846138, 0.03429795429110527, 0.6349816918373108, 0.725259006023407, -0.11915608495473862, 1.6429868936538696, 0.2033667266368866, -0.6053658127784729, 0.6095095276832581, 0.0300432201474905, 0.07493474334478378, -0.49596384167671204, -0.006136185489594936, 1.4461270570755005, 0.6016535758972168, -0.4526689648628235, -0.8125128746032715, 0.9397223591804504, -0.6540358066558838, -1.269324541091919, -0.5557560324668884, -1.8146007061004639, -0.18465739488601685, 0.8580767512321472, -1.201803207397461, 1.2010921239852905, 1.6966040134429932, -1.1656138896942139, 0.8081231117248535, -1.4108645915985107, 0.7664504051208496, 0.7692676782608032, -1.2426003217697144, -0.813480019569397, 0.3464994728565216, -1.0452901124954224, 1.5354830026626587, 1.1321107149124146, -1.5151360034942627, -1.0736831426620483, 0.08881627768278122, -0.017503606155514717, 0.9270004630088806, 0.40082767605781555, -0.1493861973285675, -0.8046698570251465, -0.8504036068916321, 0.11162672191858292, 1.0369291305541992, 0.22505466639995575, -0.6771946549415588, 0.6327235102653503, 1.4880669116973877, -0.33737924695014954, -0.48427215218544006, 0.7436746954917908, -1.3902162313461304, 1.0242737531661987, -1.0413048267364502, 0.8770943284034729, 1.1879626512527466, 0.2571726441383362, 0.6584119200706482, -0.44044139981269836, 1.9286524057388306, -0.2993099093437195, 0.8724402785301208, -0.3398326635360718, 0.5952857732772827, -0.6030834317207336, -0.5251323580741882, 0.20359362661838531, -1.033608078956604, 0.33208248019218445, 0.9491437077522278, -0.634003221988678, -1.2958050966262817, 0.0993151068687439, -0.3133626878261566, -0.255134642124176, -1.286022424697876, 1.302546501159668, 1.0859496593475342, 1.4511905908584595, -1.4274570941925049, 1.1008557081222534, 0.3125951886177063, -0.5875798463821411, -0.7601160407066345, -1.1431816816329956, 0.9296273589134216, -1.1805325746536255, 0.7977399826049805, -0.3374831974506378, 1.5416170358657837, 1.5755571126937866, -1.2635393142700195, -0.17906807363033295, 0.3466796875, -0.1456289142370224, -1.3698277473449707, 1.0234992504119873, 0.37493082880973816, -0.6486233472824097, -0.2858634293079376, 0.20568840205669403, -0.09093759208917618, -0.05659116804599762, 0.20670802891254425, 1.6956877708435059, 1.346667766571045, 0.08116963505744934, 0.6406667828559875, -0.03508681058883667, 0.9966786503791809, -1.690110683441162, -0.06993085891008377, -0.9380368590354919, 0.3569401800632477, 1.6226695775985718, -0.895207405090332, 1.2642691135406494, 2.345120906829834, 1.1959704160690308, -0.38591691851615906, 0.15298789739608765, -1.741254448890686, -0.10485576093196869, 0.007776022423058748, -1.304263949394226, -1.1515347957611084, 0.7514898180961609, -1.6508747339248657, 1.21816086769104, 0.4399050176143646, -0.25033822655677795, -1.2298699617385864, 0.42673978209495544, 1.318720817565918, -0.9172452688217163, -0.15373040735721588, 0.34098485112190247, -0.4770801365375519, 0.6628121137619019, -2.041203498840332, -0.29956579208374023, -0.9884835481643677, -0.8798546195030212, 2.378554344177246, 0.14661256968975067, 0.9612174034118652, 0.8732926249504089, -0.16899481415748596, -0.5313680768013, -0.5898545384407043, -1.5120391845703125, -0.5720356106758118, -0.290666401386261, 0.05688720569014549, 1.510834813117981, 1.0291006565093994, 1.8085376024246216, 0.8761656880378723, 0.7912080883979797, -1.2038270235061646, 0.8234778642654419, 0.5461574792861938, 1.0897479057312012, 0.03268447518348694, 0.5970742106437683, -2.7078709602355957, -0.5351106524467468, -1.0528895854949951, 0.18491901457309723, -0.5924335718154907, -1.0782874822616577, -0.4718092978000641, 2.231419324874878, 0.1550387442111969, -0.8135803937911987, -0.5430611371994019, 0.7371612191200256, -0.17373666167259216, -1.523447036743164, 1.1672260761260986, 0.42606016993522644, 0.8713973164558411, -1.576131820678711, 0.11905970424413681, 2.075658082962036, 3.0316896438598633, -0.5596910119056702, -0.5446468591690063, 0.6662846803665161, -0.7966771721839905, -0.7092341780662537, -0.10939498245716095, -1.1782714128494263, 0.10030674934387207, 0.6445621252059937, 1.8820974826812744, 0.5896292924880981, 0.26303353905677795, 1.0260615348815918, -0.7277291417121887, -0.6712663769721985, 0.7021755576133728, 0.7393468022346497, 0.06826315820217133, -0.13013771176338196, -0.29440492391586304, -1.0984965562820435, 1.2900904417037964, 0.4915746748447418, 0.7632908225059509, 0.4553660750389099, 0.45420676469802856, 1.148473858833313, 1.086672306060791, 0.6644672751426697, 0.9077790379524231, -0.5443204045295715, -0.16151097416877747, 0.9059404730796814, -0.7863644361495972, -0.7033421993255615, -0.120356485247612, 0.2574394941329956, 0.490701824426651, 0.11925626546144485, 0.7518768906593323, -0.05401340126991272, -1.2707821130752563, -0.9206142425537109, -0.6480534076690674, -1.520778775215149, -0.35707923769950867, 0.695309579372406, -0.6215025782585144, 0.10756811499595642, -1.181411862373352, 0.4542309641838074, 1.0721439123153687, 1.8916444778442383, 0.7138051986694336, -0.0058025154285132885, 1.9573546648025513, 1.0320991277694702, 0.9779654741287231, -0.576327383518219, -1.61001455783844, -0.2837122082710266, 0.6098860502243042, -0.7804965972900391, -0.40101081132888794, 1.2066118717193604, 0.5745715498924255, -2.1793951988220215, -0.19776347279548645, 0.7311686873435974, -1.0434463024139404, 0.13956068456172943, -1.634629487991333, 0.43084871768951416, 1.4220046997070312, 0.17214377224445343, -0.4386672079563141, -1.1318541765213013, 0.5487843751907349, -0.5780477523803711, 0.47821110486984253, 0.24358703196048737, -0.9521410465240479, 0.3725864291191101, 1.102827548980713, 0.9942935109138489, -0.36686277389526367, -0.24727760255336761, -0.2276456505060196, 0.5704084038734436, 0.4465189278125763, 0.35178786516189575, 0.806384801864624, -0.3049411475658417, 0.08846700191497803, 0.2884354293346405, -0.4946236312389374, 0.9913702607154846, -0.11356595158576965, -0.38969820737838745, -1.644208550453186, -0.3427105247974396, 2.9057488441467285, 0.09606882929801941, -0.06718049198389053, -0.569789469242096, 0.19977614283561707, 0.07121571898460388, 1.5698513984680176, 0.012422090396285057, -0.4984655976295471, 0.8554284572601318, 0.6482555866241455, -0.6306765079498291, 0.35764458775520325, 0.08729997277259827, 0.2533906400203705, 1.3152490854263306, 0.13451534509658813, 1.3674662113189697, -0.7665268778800964, 0.9011574983596802, -0.4601965546607971, 1.025854229927063, 0.49400201439857483, 1.2416598796844482, -0.5944241285324097, -0.6927459239959717, 0.30422839522361755, -0.20764577388763428, -2.0049216747283936, 0.8238590955734253, 0.22894717752933502, 0.6148720383644104, -1.713761568069458, 0.03402918204665184, -1.73924720287323, -0.5436087846755981, 0.4904087483882904, -1.0645740032196045, -1.926088809967041, -1.0583124160766602, 1.0124894380569458, 0.09273524582386017, -0.9093309640884399, 1.3742812871932983, 0.853959321975708, -0.26223260164260864, -0.03847203031182289, -0.3820289969444275, 1.0767935514450073, 0.9271990656852722, 0.614892840385437, -0.47787749767303467, -0.034913498908281326, 1.0209624767303467, 1.3153778314590454, -1.5713353157043457, 0.3817512094974518, -0.9618418216705322, -0.003119424683973193, -0.32407233119010925, -0.2207627147436142, 0.980874240398407, -0.05203134939074516, 0.1326727718114853, 1.151843786239624, -0.3555878698825836, 1.1906116008758545, 1.031574010848999, -0.8441055417060852, -0.6308764219284058, -0.009628169238567352, 0.6214519739151001, 0.7145813703536987, 0.6848750114440918, 0.8588157892227173, 1.222313642501831, -2.0544278621673584, -0.3034548759460449, 0.18058262765407562, 0.1772192269563675, 0.47737908363342285, -0.03369232267141342, 1.0169984102249146, 0.07057437300682068, -0.08407232165336609, 0.32629066705703735, -0.04532041773200035, -1.0961520671844482, 1.4987986087799072, -0.12284183502197266, -1.6428459882736206, -0.4496658444404602, 1.8837183713912964, 0.12334030121564865, 1.7388228178024292, 1.1920266151428223, -1.4097883701324463, -0.5292391180992126, 0.0033114321995526552, 1.9092018604278564, -0.20573699474334717, 0.47393861413002014, 0.3302578628063202, 0.28224053978919983, -0.046908047050237656, -1.5180516242980957, -0.30929824709892273, 0.5226697325706482, -1.2592054605484009, 0.1120886579155922, -0.1236819252371788, -0.9207674860954285, -1.019161343574524, -0.19854256510734558, 0.09248585999011993, -0.4111533463001251, -1.8959037065505981, 0.07792028784751892, 0.5976548194885254, -1.364220380783081, 0.21815799176692963, 1.4675390720367432, -0.9385747909545898, -0.14549177885055542, 1.3179209232330322, 2.091949462890625, -1.3301490545272827, -0.49648210406303406, -1.670719861984253, 1.2792835235595703, 2.2945942878723145, -0.4655662477016449, -0.20336298644542694, -0.4247365891933441, 1.348273515701294, -0.9899522066116333, -0.34887823462486267, -0.09368343651294708, -0.7351277470588684, -2.0768980979919434, -0.7392659783363342, -0.1702132374048233, 1.1671369075775146, -0.23287634551525116, 1.2844992876052856, 0.14660537242889404, -1.4827617406845093, -1.0212262868881226, 0.667487621307373, -1.3054091930389404, -1.7433710098266602, 1.4826725721359253, 0.1938895881175995, -0.7163153290748596, -0.1826280802488327, 1.172852635383606, 0.5539495348930359, -0.09756627678871155, 0.036016903817653656, 0.22105897963047028, 0.6890050172805786, -0.7858998775482178, 0.4149940609931946, -1.7655961513519287, 2.2487285137176514, 0.07706056535243988, -0.6498534083366394, -0.6701462268829346, -0.9633112549781799, 0.3636007308959961, -0.3787633180618286, 0.1990562379360199, -0.04468127340078354, -0.37111687660217285, -1.7663975954055786, 0.29155150055885315, -0.04008346423506737, 0.036667726933956146, -0.2745089530944824, -0.9667448997497559, -0.42113760113716125, -0.7818511724472046, -0.16279882192611694, -1.5382452011108398, -0.5167831182479858, 1.154131531715393, 1.347511887550354, 0.1818036437034607, 0.7572113275527954, -0.09735362976789474, 1.00050950050354, -0.144289031624794, -0.574608564376831, 0.3421361446380615, -0.5798060297966003, -0.6397911906242371, 1.0146020650863647, -0.870741069316864, 1.5240509510040283, 1.2119555473327637, 1.2444196939468384, 0.890525221824646, -0.6158682107925415, 0.36556804180145264, -0.8264731168746948, 0.5999065041542053, -0.7309675216674805, 0.40564432740211487, 0.4495016932487488, 0.32823309302330017, -0.7505772113800049, 0.5903186798095703, 0.674828052520752, 0.7908177971839905, -1.345991611480713, 0.9483262300491333, 0.6970853209495544, -0.24517489969730377, 0.9838131070137024, -0.18347421288490295, -1.1204265356063843, 0.4479066729545593, 0.6647679209709167, 0.4289875030517578, 0.72035813331604, 0.6154014468193054, -0.7751173973083496, -0.25055959820747375, 0.25773128867149353, -0.306468665599823, -1.701568603515625, 0.3122657537460327, 0.24920028448104858, 0.5247969031333923, 0.7366610765457153, 0.3494439125061035, 0.17790989577770233, -1.2734802961349487, 0.03179465979337692, -0.5001946687698364, 0.13155095279216766, 1.082038164138794, -0.5695585608482361, 0.462005615234375, 1.1428358554840088, -0.2683429419994354, 0.2723528742790222, 0.9378149509429932, 0.6242454648017883, 1.430545687675476, 0.6075674295425415, 1.0766360759735107, 1.183991551399231, 0.2409815490245819, -0.3408360481262207, -1.2237277030944824, 1.0163531303405762, 1.1560065746307373, 0.4400102198123932, -0.35145503282546997, -0.503157377243042, -2.075591802597046, -1.1069329977035522, -1.5159275531768799, -1.941557765007019, -2.276568651199341, 1.182186245918274, 1.2902978658676147, -0.1436314582824707, 0.03162286803126335, 0.3350159823894501, 1.409780740737915, 0.25212764739990234, 0.6538326144218445, 1.458349347114563, -0.3356739580631256, -0.27097752690315247, 1.1267125606536865, -0.06810305267572403, 1.5312241315841675, 0.3917911946773529, -0.36826688051223755, 3.00520920753479, -0.35152697563171387, -0.16874653100967407, -0.5697237253189087, 0.8023883700370789, 0.649303674697876, 0.1315019726753235, 1.6639467477798462, 0.15663981437683105, 0.32559019327163696, -1.1481364965438843, 0.6719515323638916, 1.686100721359253, 0.6642706394195557, -1.9177885055541992, -0.5625554323196411, 0.7038296461105347, 0.12709859013557434, -1.31563401222229, 0.14476105570793152, -1.5316523313522339, -0.7126926183700562, 1.7062225341796875, 0.4613425135612488, -0.5469772219657898, 1.5018755197525024, -0.030722390860319138, -1.0354034900665283, -2.76781964302063, -1.0045148134231567, -0.42552709579467773, -0.3634474575519562, -1.9291936159133911, 1.4063469171524048, 0.36167219281196594, 0.3521033823490143, -0.3651129901409149, 0.4950483441352844, 0.5458473563194275, 1.259484052658081, 1.418251395225525, 0.6499295234680176, -1.5980945825576782, 0.3959084153175354, 0.7337874174118042, 1.4143272638320923, -1.1577699184417725, 1.657913088798523, 0.23360657691955566, 0.9822088479995728, 1.0074687004089355, -0.2964918911457062, 1.0010945796966553, -0.42073604464530945, 0.697816014289856, -2.278243064880371, 1.8766793012619019, 0.020938610658049583, 0.5326528549194336, 0.6345081925392151, 0.035319723188877106, -1.7084670066833496, -1.2941383123397827, 1.4608830213546753, -0.5963067412376404, 0.7315176129341125, 1.8749914169311523, -1.1390973329544067, 1.5717023611068726, 2.11240816116333, 0.2151138335466385, 0.2008167952299118, 0.3720044791698456, -1.4168546199798584, 0.9719452261924744, 1.5233285427093506, -1.5778571367263794, -0.21501697599887848, 2.3517587184906006, 1.6177773475646973, -1.0110969543457031, 3.215156078338623, 1.3217027187347412, 1.5213874578475952, -0.6203796863555908, 0.04052167385816574, 1.9391114711761475, -0.5470190644264221, -0.1805686056613922, 1.448399305343628, -0.6078187823295593, -1.3299075365066528, 1.2635993957519531, -2.2164454460144043, 0.7667713165283203, -2.1356968879699707, -0.1805511862039566, 0.7505267262458801, -2.689340829849243, 0.12788088619709015, -0.4438210129737854, 0.017210861667990685, -0.8025625348091125, 0.06934147328138351, 0.23720532655715942, 0.033297546207904816, -0.020577482879161835, 2.4270122051239014, 0.6257541179656982, 0.5386119484901428, -2.147717237472534, -0.14629806578159332, -0.34247878193855286, 0.041527193039655685, -0.11845976114273071, -0.6102151870727539, 0.070022352039814, -1.0048173666000366, 1.0300326347351074, 1.0477668046951294, -0.5096426606178284, -1.0201175212860107, 0.092457614839077, 0.27395567297935486, -0.1949211210012436, 1.0694947242736816, 1.45296049118042, -0.8944472074508667, 2.016713857650757, -0.13750876486301422, -1.6740803718566895, -0.8002824187278748, 0.5698859095573425, -2.31400990486145, 0.8693297505378723, 2.854907512664795, 0.854476273059845, 0.5476510524749756, 1.0641231536865234, -0.2416587471961975, -1.7940315008163452, 0.66643226146698, -1.0032553672790527, -1.24240243434906, 0.7414290904998779, -1.64285147190094], [0.17015637457370758, 0.30566734075546265, -1.0033626556396484, -1.0173587799072266, -0.6879944205284119, 0.7848531603813171, 2.688858985900879, 0.8609275817871094, 0.8401210904121399, -0.6050317883491516, -0.661370038986206, 1.5079386234283447, 0.33484676480293274, -0.07845001667737961, 0.4449234902858734, -0.03768214210867882, -0.13410048186779022, -0.592135488986969, -1.855904459953308, 0.5704926252365112, -0.9890735745429993, 1.0356581211090088, -2.5553183555603027, 0.012967928312718868, 1.470503807067871, -0.9297932386398315, -0.7320848703384399, 0.21031485497951508, -0.17144449055194855, 0.08331991732120514, 1.202369213104248, -0.8835093975067139, 1.5367697477340698, 1.1430243253707886, 0.053361937403678894, -0.41029825806617737, 0.6148483753204346, -0.48312726616859436, -1.1548421382904053, -0.5152359008789062, -1.192278265953064, -1.806380271911621, -1.3851916790008545, -0.22669106721878052, -0.1049349457025528, -2.1663761138916016, 0.052386198192834854, 1.9146804809570312, -0.9158234596252441, -2.4022347927093506, -2.9735007286071777, 0.24691833555698395, 1.7681316137313843, -0.9926974773406982, 0.45054349303245544, 1.6708118915557861, -0.15800681710243225, 0.2273630052804947, -0.7004989385604858, 1.1931203603744507, 0.46861687302589417, 0.5166090130805969, 1.169639229774475, 0.2655665874481201, -0.11628947407007217, -0.28303346037864685, -1.124206781387329, -0.028730623424053192, 1.154691457748413, -1.505571961402893, -1.006956696510315, -1.881211280822754, -1.614422082901001, -0.856647789478302, 0.8676811456680298, -0.7575861215591431, -2.02616286277771, 0.34606677293777466, -1.709557056427002, -0.15779462456703186, 1.0155822038650513, 1.6061768531799316, -0.06768631935119629, 0.5652390718460083, -0.08278211206197739, 0.5804972052574158, 0.2658332884311676, -1.7351707220077515, -1.7397757768630981, -0.2336280345916748, 1.0610796213150024, -0.2801123261451721, 1.5229686498641968, 1.309753656387329, -0.8052955865859985, 1.0663459300994873, -0.7173663973808289, -2.006171703338623, -0.9512303471565247, 0.9645917415618896, -0.17514978349208832, -1.6007027626037598, -0.4542255401611328, -1.7755732536315918, -0.35383760929107666, -0.33489862084388733, 0.4145604372024536, 0.34853920340538025, 0.2920774519443512, 0.03838106617331505, 0.5034247636795044, 0.8709745407104492, 2.7314164638519287, -1.044329285621643, -1.8030979633331299, 0.42975443601608276, 0.07487281411886215, -0.7893881797790527, -0.1984899491071701, 1.3641592264175415, 0.6322388052940369, 1.7197239398956299, 0.5224982500076294, 1.157317876815796, -0.23766270279884338, -0.24932818114757538, -2.3596503734588623, 0.3326931297779083, 0.42155689001083374, -0.6587426662445068, -0.09469939023256302, 0.22636312246322632, 1.8993134498596191, 0.6290645599365234, 0.7337038516998291, 0.4551243484020233, 0.07586094737052917, 0.5238409042358398, 0.5115338563919067, 0.8948043584823608, -0.6093053221702576, 3.0934178829193115, -0.38810840249061584, 1.221261739730835, -1.8069164752960205, -1.4145638942718506, 0.4619195759296417, -0.6738523840904236, 2.568807601928711, -0.5655611157417297, 0.694274365901947, -0.20333483815193176, -0.9627326130867004, 1.5162177085876465, 0.32997238636016846, 0.4502653479576111, -1.5240960121154785, 0.9172064661979675, -0.429983913898468, 0.9160833954811096, -1.3928806781768799, -0.2815017104148865, -0.09016416221857071, -0.5899143218994141, 0.3407672345638275, -0.4775988757610321, -0.6545211672782898, -0.7908933162689209, -0.20805007219314575, 0.22810572385787964, 0.3506641685962677, 1.8107929229736328, 1.7134042978286743, -0.25974854826927185, 2.79892897605896, -0.21101278066635132, 0.1313258707523346, 0.43548232316970825, -2.1122279167175293, -1.0309090614318848, 1.080331802368164, 1.9528332948684692, -0.2582353949546814, 1.1860036849975586, 0.15449213981628418, 0.17625419795513153, -0.232753723859787, -1.1007685661315918, 0.056726932525634766, -1.058975100517273, -2.1066081523895264, -0.5920655727386475, -0.1226806715130806, -0.7287634015083313, 0.2570790648460388, -0.40172284841537476, -1.2440450191497803, 0.4113357961177826, -0.38018301129341125, -0.014461598359048367, 0.22864975035190582, 1.1244648694992065, 0.3178504407405853, -2.340998649597168, -0.5238615274429321, 1.7801073789596558, 0.06716839224100113, 0.6023107767105103, -0.09138624370098114, 0.17594581842422485, -0.6455155611038208, 1.6069122552871704, -1.9063780307769775, -0.5452483892440796, 0.9387418031692505, -1.4603530168533325, -0.6514860391616821, -0.4781394600868225, 1.1760576963424683, 0.007891393266618252, 0.32550284266471863, 1.2405478954315186, 0.8057448267936707, 0.4723915755748749, -0.9332334399223328, -1.1182211637496948, -2.1689608097076416, 0.46017709374427795, 1.8843551874160767, -1.224525809288025, -0.6715370416641235, 0.18842178583145142, 0.3147481381893158, -0.07894810289144516, 0.5858411192893982, -1.1713058948516846, 1.2336591482162476, -1.0561347007751465, 0.25841400027275085, -0.6836778521537781, 0.31097209453582764, -0.0035045677796006203, -0.5769566297531128, 0.11450759321451187, -1.551879644393921, 0.8012687563896179, -0.3971925973892212, -1.340383529663086, 0.053754277527332306, -2.7866063117980957, 0.11208322644233704, -0.7041718363761902, 0.7435582280158997, -0.2874608635902405, -1.7330716848373413, 1.099066138267517, -1.3182778358459473, -1.0345338582992554, -0.059371158480644226, -0.7585359215736389, 0.39023977518081665, 1.255340814590454, -0.8890408277511597, -1.320688009262085, -1.7029550075531006, -0.7309703230857849, -0.4988034963607788, 0.8374091982841492, 0.4611753821372986, -0.45904290676116943, -1.6416780948638916, 0.7750606536865234, 0.6360867619514465, 0.7382087111473083, 1.2174043655395508, -1.7963393926620483, -1.0669270753860474, 0.8089593648910522, 2.2214159965515137, -1.1230144500732422, 0.45961281657218933, -1.0482988357543945, 0.04621158167719841, 0.017223207280039787, -1.1206446886062622, -0.47349220514297485, -0.20238150656223297, -0.5765831470489502, 0.698208749294281, 1.0288134813308716, -1.4988652467727661, 2.9147419929504395, 0.27910947799682617, -0.08967111259698868, 0.2745039165019989, -0.2169179916381836, 1.3541483879089355, -0.16101165115833282, 0.3090759813785553, -0.20047113299369812, -1.165157675743103, -0.4671916365623474, 0.6046915054321289, 1.7017796039581299, 0.7692446708679199, 1.827661395072937, 0.7896631360054016, 0.40140819549560547, 0.0739053413271904, -0.9103720784187317, 1.286181926727295, 1.3019531965255737, 0.231284499168396, -0.2763841152191162, -0.4255576431751251, 1.570112943649292, 1.563291311264038, -0.5410386919975281, -0.5900410413742065, -1.1934078931808472, 0.7528032064437866, -0.7701773643493652, 1.587648630142212, 0.2655790448188782, -1.195082426071167, -0.3195815086364746, 1.0392712354660034, 1.960771918296814, 2.1407968997955322, 0.517157256603241, 0.9930331707000732, 1.0107141733169556, -1.7195444107055664, -1.121856927871704, 0.2529822289943695, -1.007086992263794, 0.7949845194816589, -0.5756516456604004, -0.345530241727829, -2.0108258724212646, 0.16245310008525848, 1.1556373834609985, 0.14027664065361023, -0.7390170097351074, -0.31148192286491394, -0.898773729801178, 0.8289588093757629, -0.5315352082252502, -1.2849260568618774, 0.3893658220767975, -0.27785855531692505, 0.8078109622001648, 0.10220105201005936, 0.20670084655284882, 0.21166084706783295, -0.17510223388671875, -0.3759043514728546, -0.9986611008644104, 1.859788179397583, 0.28852415084838867, -0.20545992255210876, 0.34136056900024414, -1.3024208545684814, -0.21474143862724304, 0.5443243384361267, -0.2470705509185791, 2.3030712604522705, -1.5792211294174194, 0.22750943899154663, -0.22783909738063812, -0.8682957887649536, -0.6171019077301025, -0.6375520825386047, 0.038152292370796204, -1.7915745973587036, -0.8258659839630127, -1.6260499954223633, 0.3631017208099365, -1.168906331062317, -0.18863621354103088, -2.4808409214019775, 0.9923738837242126, 1.2177857160568237, -0.07087888568639755, 0.6970303654670715, 0.776047945022583, 0.3203500509262085, -1.4200328588485718, 0.24219584465026855, 0.3573083281517029, 1.355201005935669, 0.11047598719596863, -1.6120386123657227, -0.2735517919063568, 0.3752621114253998, 1.2719184160232544, -0.20892703533172607, 0.3843260705471039, 0.7787865400314331, 0.4740099012851715, 1.4352763891220093, -0.6902422904968262, -0.4160206615924835, 0.1115168109536171, 0.4762813448905945, 0.43457791209220886, -0.4380025565624237, 1.6604496240615845, 0.8070467114448547, -1.396695613861084, -0.22765642404556274, -0.6867877244949341, -0.4001624286174774, 0.7549785375595093, -0.09732180088758469, 1.6637109518051147, 1.0385465621948242, -0.20880751311779022, -0.15140287578105927, 0.37817439436912537, 0.11540257185697556, -0.5061152577400208, 1.334981918334961, -0.023763511329889297, 0.9150969982147217, -1.7376635074615479, 0.8523463606834412, 1.5109964609146118, 1.258112907409668, -0.06632659584283829, 0.4604904055595398, 0.9651684761047363, 0.813440203666687, -1.2303557395935059, 0.4146367907524109, -0.8360980749130249, 1.7568295001983643, 1.3399282693862915, -0.7175893783569336, -1.019606590270996, 1.5350295305252075, 0.434770792722702, -0.6051504611968994, 0.04882922023534775, 0.8091627955436707, -1.03304922580719, 1.0854374170303345, 0.5176517963409424, 0.314481645822525, 1.1706664562225342, 0.7404772043228149, -1.2558820247650146, 0.8443711996078491, 1.7272841930389404, -1.0598222017288208, 0.13865210115909576, -0.7484183311462402, 0.864787220954895, -0.7935208678245544, -1.2896511554718018, 0.8678189516067505, 1.4836546182632446, -0.23353369534015656, 0.381692498922348, 0.5355759859085083, 0.41823452711105347, -0.7149562835693359, 2.3578593730926514, 0.8043699264526367, -0.9212427735328674, -0.6364151239395142, -0.4752960503101349, 0.6248927712440491, 0.5024728178977966, 0.22803418338298798, -1.1118006706237793, 1.8685814142227173, -1.464789628982544, 0.37550869584083557, -1.4024008512496948, 0.9901961088180542, -1.1098225116729736, -0.1934846043586731, -0.2169782966375351, 0.1484285295009613, -1.3661659955978394, 1.4283467531204224, 0.1396600902080536, 0.7359218001365662, 0.05807371810078621, 0.2832411527633667, -1.4404079914093018, -2.281750202178955, -0.6125944256782532, -0.1118292286992073, -0.33922576904296875, 1.6480393409729004, -0.8749462962150574, -2.154930830001831, -1.2488453388214111, -0.8364271521568298, 0.0005322173237800598, 0.6618523597717285, 0.5122527480125427, 1.1202080249786377, 1.1069353818893433, 0.5164427161216736, 1.017242670059204, 0.9943615794181824, 0.41563281416893005, -1.7331260442733765, -0.6511450409889221, -0.5143446326255798, -1.1822404861450195, 0.989630937576294, -0.47927963733673096, 0.2823061943054199, 1.5498720407485962, -1.432963252067566, 0.3823096454143524, -0.3800398111343384, -0.2514631450176239, 0.6684166789054871, -0.38911348581314087, 1.1271450519561768, -0.6954394578933716, -0.11319322139024734, 1.0805652141571045, -0.26383402943611145, 0.763939619064331, 0.4444338083267212, 0.25478100776672363, -2.3359568119049072, 0.05100395902991295, -2.4841859340667725, -5.8892332162940875e-05, 0.5501972436904907, -1.7046574354171753, -0.7133060693740845, -1.9523155689239502, -1.0264863967895508, 0.7756473422050476, 1.45857834815979, -0.344845175743103, 0.6792066097259521, 1.517229437828064, 0.7405158281326294, -1.398337483406067, 0.18758149445056915, 0.4957609176635742, -0.10742016136646271, -1.0833940505981445, -0.5055946111679077, -0.5923165082931519, -1.1237553358078003, -0.28123244643211365, 0.30305036902427673, -1.5472064018249512, 1.1816493272781372, 1.82504141330719, -0.3834693431854248, 0.6461565494537354, -0.1666272133588791, 2.399015426635742, 0.5530440807342529, 0.10878763347864151, 0.9545092582702637, 0.20464660227298737, -0.20301729440689087, 1.0559108257293701, -0.7753587961196899, 0.13908791542053223, 0.6568521857261658, 1.2406539916992188, -0.013919789344072342, 1.1795361042022705, -0.2934797704219818, -2.33852481842041, -0.020467033609747887, 1.2004584074020386, 0.6906468868255615, 0.9040468335151672, -1.014311671257019, -0.6013925671577454, -1.5166277885437012, 1.7510820627212524, -0.8338820338249207, -1.3176599740982056, 0.07015378773212433, -2.349992275238037, 1.8960050344467163, -1.0489286184310913, 1.285506248474121, -1.4923405647277832, -0.4217079281806946, 1.114623785018921, -0.6573371291160583, 0.12833508849143982, -0.551450252532959, -1.4858057498931885, 0.15263819694519043, 0.8771982789039612, -0.2833654284477234, 0.5257282853126526, 0.07611624896526337, -0.6543231010437012, -1.096501350402832, 0.7833360433578491, 1.4182325601577759, 0.5861585140228271, 1.6364177465438843, -1.141116976737976, 0.761989414691925, 2.753309965133667, 0.6398760676383972, 0.46308228373527527, -1.1784071922302246, -0.9096774458885193, 1.671623945236206, 0.20361727476119995, -0.3176330029964447, 0.3930909037590027, -3.0188493728637695, -0.14600180089473724, -1.3677116632461548, -0.010584860108792782, -1.1420296430587769, 0.11413071304559708, 0.6702641844749451, -0.9406724572181702, 0.16700759530067444, 0.6761335134506226, -0.3515539765357971, 1.4755209684371948, 0.9442979097366333, -0.4311395287513733, -1.3966439962387085, 0.44192561507225037, -0.628311276435852, 2.044320583343506, -2.1170151233673096, 0.5748117566108704, 0.8198032975196838, -1.1541404724121094, -0.9317144751548767, -0.8453145027160645, -0.18117080628871918, -0.019505897536873817, 0.6427105665206909, 0.56907719373703, 0.19037023186683655, -1.223111629486084, 0.26089590787887573, 1.2447923421859741, -0.30610156059265137, -1.0149810314178467, 0.7271921038627625, 0.5921953916549683, -1.4588942527770996, -1.0352802276611328, -1.172038197517395, 0.42076414823532104, 0.897908091545105, -2.5064213275909424, -0.8895900249481201, 0.8045862913131714, -0.34448885917663574, 0.38869643211364746, 0.40314438939094543, -1.6364359855651855, 2.0976200103759766, -0.6697974801063538, 0.30909475684165955, 0.5499786734580994, 0.08112480491399765, 0.3720165193080902, -0.31051817536354065, 0.8657752871513367, 1.45823073387146, 0.2621889114379883, -0.25817975401878357, -0.46561241149902344, 0.6204879283905029, 0.45481181144714355, 0.2785714864730835, 0.6440566182136536, -0.7227470874786377, 0.2957688868045807, -0.27368906140327454, -2.0199456214904785, 0.9150301814079285, -0.940095841884613, 1.411230444908142, 0.07346918433904648, -0.6535681486129761, 0.5969470739364624, 0.8866767287254333, -2.664114475250244, -0.6943434476852417, 0.20690494775772095, -0.004018026404082775, -0.189374178647995, -0.785618007183075, -1.7755995988845825, -0.30757683515548706, 1.4150227308273315, -1.0411542654037476, -0.04117308184504509, -0.22813092172145844, 1.7657783031463623, 2.414893865585327, -1.5643097162246704, 0.3762093186378479, -1.384452223777771, -0.10694155097007751, 0.9996183514595032, -1.917262315750122, -1.3100814819335938, -0.09131357073783875, -0.7140419483184814, 0.44296711683273315, 0.755485475063324, -1.8716834783554077, 0.8784253001213074, 1.0472770929336548, 1.0678799152374268, 0.37527674436569214, -0.40149208903312683, 0.9062237739562988, 1.3658345937728882, -0.13723446428775787, -1.6471045017242432, -1.630189299583435, -0.3330830931663513, 0.3190523087978363, 1.3433680534362793, 1.3340634107589722, 0.43537312746047974, 0.11215348541736603, 1.2431073188781738, 1.1672406196594238, -0.569660484790802, 0.3023080825805664, 0.15893018245697021, -0.4925621449947357, -1.3310518264770508, -1.0890538692474365, -1.1605478525161743, 0.6170439124107361, 0.3615863621234894, 0.615318238735199, 0.34585729241371155, -0.8177976012229919, -0.3338008224964142, 1.260542631149292, 0.2437412589788437, -2.1943349838256836, -1.4760538339614868, 0.17050005495548248, -1.734954595565796, 0.27150923013687134, -0.10060857236385345, 0.25112324953079224, 0.6022898554801941, 0.8911701440811157, 1.8225425481796265, -0.36541086435317993, 0.14207205176353455, 0.8134728074073792, -0.7216692566871643, -0.18186752498149872, 0.4354587495326996, -1.0569946765899658, -0.5663389563560486, -0.6782680153846741, -2.009561777114868, 0.5796153545379639, 0.44318345189094543, 0.5576952695846558, 0.32353395223617554, -0.4885227084159851, 1.1523463726043701, 0.23901362717151642, 1.3475748300552368, 0.6345285177230835, 0.41866856813430786, 0.03111281804740429, 0.15396583080291748, 1.5639351606369019, -0.7304940819740295, 1.6981635093688965, -0.9025830030441284, -0.7547072768211365, -0.6774799823760986, -0.2467365860939026, 0.9542014002799988, -0.8160452246665955, -2.3694262504577637, 0.5415366888046265, 2.454857110977173, -0.14541907608509064, 1.7343811988830566, 0.3226737678050995, 0.9509819746017456, 0.8257859349250793, -1.6152458190917969, -0.44147902727127075, -0.0627744272351265, -0.0034671444445848465, -0.24363330006599426, 1.893286108970642, -0.013258298859000206, -0.06862195581197739, -1.4151092767715454, 1.827506184577942, -1.9448132514953613, 0.5029312968254089, 1.0159568786621094, -1.3414690494537354, -0.8371008634567261, -1.0917006731033325, 0.5686318278312683, 0.3080349564552307, -1.0184259414672852, 0.39839091897010803, 0.9517343044281006, -0.34907016158103943, 0.6534029841423035, -1.7147464752197266, 0.2481764405965805, -0.28400298953056335, 0.16709376871585846, 0.9947414994239807, 0.9756449460983276, -1.2423100471496582, -2.561471700668335, 1.6529484987258911, -1.4161547422409058, 0.9054567813873291, -0.23565010726451874, -1.3986555337905884, -0.8084520697593689, -1.7365002632141113, -1.8260695934295654, 0.912713348865509, 0.4147002398967743, -1.780134916305542, -0.751910924911499, -0.8698862791061401, -1.4456006288528442, 0.16726505756378174, 0.7586031556129456, -1.6834214925765991, 1.1465789079666138, 0.16890904307365417, 0.772186815738678, -0.0870385617017746, -1.734190821647644, -0.24039126932621002, -1.159683108329773, -1.932924509048462, 0.44803521037101746, 0.5780391693115234, -1.0217502117156982, 0.5442284941673279, 0.7028557658195496, -0.9538537263870239, 1.3256863355636597, -0.9100069403648376, -0.221864715218544, 0.5434167385101318, 0.6437113881111145, -0.3827819228172302, -0.535603940486908, 0.8049777150154114, -1.347551703453064, -0.11181843280792236, -0.5174206495285034, 1.522903323173523, 0.8108741641044617, 2.522726535797119, -0.9439447522163391, 0.23711179196834564, 0.12927652895450592, -0.2679833471775055, 1.618428111076355, -0.09038958698511124, -1.6556693315505981, 0.5889137983322144, 0.1524510681629181, 0.7717118859291077, 2.05837345123291, 0.8212727308273315, 0.2264583706855774, -1.1589261293411255, -0.4508729875087738, -0.19083695113658905, -0.10298432409763336, 2.179305076599121, 2.0915865898132324, 0.011866218410432339, -0.15942589938640594, 0.24286651611328125, -0.6379002332687378, -1.4973558187484741, 0.0024504312314093113, -1.24573814868927, -0.09777770936489105, 0.5450049042701721, -0.4008375108242035, -1.1767568588256836, -1.1697064638137817, 0.11873845010995865, 1.3016868829727173, -0.009619073942303658, 1.005365252494812, -0.5185734033584595, -0.913318395614624, 1.0393208265304565, 0.5162886381149292, -1.8348478078842163, -0.8559751510620117, -2.103720188140869, -0.33713582158088684, -0.45299291610717773, 0.8740018010139465, -0.324539452791214, -0.0034846002236008644, -0.04856490716338158, 0.01872382126748562, 0.4932209551334381, -0.08101075142621994, -0.7029969692230225, -1.0573774576187134, 1.1079243421554565, 0.3340111970901489, -0.19111819565296173, 0.6037229895591736, -0.2464723140001297, 0.2504636347293854, 0.1627970188856125, -0.05475836992263794, -1.1151766777038574, 1.0708805322647095, 0.7236151695251465, -0.40524712204933167, -0.01730360835790634, 2.0245249271392822, 0.31878989934921265, 0.47509732842445374, -0.08772113174200058, -0.22267983853816986, 0.021756092086434364, -0.23764581978321075, -1.4931079149246216, 1.4911452531814575, 1.3237557411193848, -0.8301404714584351, -0.2560540437698364, 0.649257242679596, 0.6727692484855652, -0.43036770820617676, -0.44745615124702454, 0.1597364991903305, -1.0747753381729126, 0.8194765448570251, 0.8714507818222046, 1.703016996383667, 1.0392295122146606, 0.2591545879840851, -2.0080785751342773, -2.6886589527130127, -0.3234076201915741, -0.7785303592681885, -0.16458681225776672, -0.48566198348999023, 0.4737497866153717, 0.2986253499984741, -1.0801655054092407, 0.9265453219413757, 1.3361254930496216, 0.2833007574081421, 0.062313079833984375, 0.07171902805566788, -2.0687522888183594, -0.06434928625822067, -2.1052520275115967, 1.9147570133209229, 0.5993036031723022, 0.8380416631698608, 1.333523154258728, 0.02572125755250454, 0.451896995306015, 1.8453757762908936, -0.4050080180168152, -0.4688488245010376, 2.46404767036438, -0.800740659236908, -0.8878198862075806, -0.5675038695335388], [-0.3958616554737091, -0.47182387113571167, -0.7218066453933716, 0.964966356754303, 3.4957518577575684, -1.7298228740692139, 0.7204537391662598, 0.47431543469429016, 0.054780445992946625, 0.129871666431427, 0.4899246096611023, 0.35236603021621704, -0.1442681849002838, 0.2721974849700928, 0.5264172554016113, -0.025792423635721207, -0.31539061665534973, -2.0567331314086914, 0.7044671773910522, -1.7129223346710205, -0.48826128244400024, -0.2913787066936493, -1.7739485502243042, -0.21687324345111847, 1.9680386781692505, 1.4992830753326416, 0.6105647087097168, 0.30697908997535706, -0.8439849615097046, 0.27956581115722656, -0.963640570640564, 0.9422364830970764, -1.5547641515731812, 0.31851986050605774, -0.4162738025188446, 0.11250001192092896, -0.5903720855712891, -0.9261291027069092, -0.17433391511440277, 0.018765226006507874, -1.4226874113082886, 0.1366821527481079, 0.03455035388469696, 0.1715507209300995, -0.03161420673131943, 0.6969617009162903, -0.4934818744659424, 0.8009209632873535, -0.4397476017475128, -0.07431568205356598, -0.3147619366645813, -1.8029022216796875, 0.40885400772094727, 2.676339626312256, 0.11830675601959229, -0.47201573848724365, -1.8852648735046387, 0.7273947596549988, -0.11635547131299973, -0.1969408392906189, 0.7009943127632141, 0.37967661023139954, 0.09664656221866608, 0.45269840955734253, 0.9740015268325806, 0.7152076959609985, -0.9176608920097351, -0.8112006783485413, -2.1730852127075195, -0.48323842883110046, 0.7143131494522095, 0.0037910067476332188, 0.3899282217025757, 0.5375022888183594, 0.4007538855075836, -0.34561628103256226, -0.8271152973175049, 0.08385181427001953, -1.4410462379455566, -0.8102666139602661, -1.4842662811279297, -1.0660309791564941, 1.000313401222229, 1.2584447860717773, -0.5294384360313416, 0.4710566997528076, 0.8277465105056763, -1.1938025951385498, -0.16305804252624512, -0.8493895530700684, -0.7491421103477478, 0.40801775455474854, 0.6046083569526672, 1.4595130681991577, -0.29220491647720337, -0.463817834854126, 0.9207772016525269, -1.1980700492858887, -0.17127443850040436, 0.1644648015499115, -0.46703577041625977, 0.6210935115814209, 0.9466100335121155, 1.1080455780029297, 0.9345291256904602, 0.7351006269454956, 0.8054201602935791, 1.7838246822357178, -0.08823317289352417, 0.8385400772094727, 0.743009090423584, -0.4008372724056244, -0.7563996315002441, -0.35510674118995667, -0.7740028500556946, -0.5296288728713989, 1.8153691291809082, -0.7072380185127258, 1.3384243249893188, 2.053255796432495, -0.04367667809128761, 0.5884448289871216, -0.7214519381523132, -1.2765480279922485, -0.8292946815490723, 1.3445351123809814, -1.0355780124664307, 0.6175305843353271, -0.4074266850948334, -0.1516779065132141, 0.725532591342926, 0.10358547419309616, 0.10633262991905212, -0.8378382325172424, 0.6726289987564087, 0.3668614327907562, 1.3894826173782349, -0.38483843207359314, -0.4539429843425751, -1.3943674564361572, -0.32946452498435974, 0.09380451589822769, 1.4705575704574585, 1.3302409648895264, 0.9987527132034302, -0.14662198722362518, 0.7404835224151611, 0.12293267995119095, -0.7318540811538696, 0.7345999479293823, 0.7442846894264221, 0.8492918610572815, 1.9029978513717651, 0.6734234690666199, 3.1193442344665527, -0.21421536803245544, 1.9554020166397095, 0.19849291443824768, 0.15899600088596344, -0.3582391142845154, -0.908822238445282, 0.5130224823951721, -0.5154138207435608, 0.4653255343437195, -1.3115439414978027, -0.7394322752952576, -0.5271250009536743, -1.6430808305740356, -1.5917353630065918, 0.4009029269218445, 0.25101372599601746, 0.2608090043067932, -0.7147674560546875, -0.19810838997364044, 0.10498311370611191, -0.877935528755188, -0.9273743033409119, 1.916175127029419, -0.011308574117720127, -0.3938557207584381, -0.8243889808654785, -0.12006624788045883, 0.3962881565093994, 0.8104552626609802, 1.2091279029846191, 0.3968788683414459, -0.685699462890625, 1.1454269886016846, 0.4579039514064789, -0.0632210522890091, -0.43777012825012207, -0.3745497167110443, 1.2617379426956177, 0.3103081285953522, 1.613731026649475, 1.0066146850585938, -0.5982248187065125, 1.0410076379776, 0.9445470571517944, -0.9198169708251953, -0.3957662284374237, -0.035178497433662415, -0.05139872431755066, -1.0165122747421265, 2.4461543560028076, -0.598334789276123, -1.5583550930023193, -0.670017421245575, -1.3593804836273193, -0.4253045916557312, -1.068169116973877, 0.8314047455787659, 0.5447595119476318, -1.1660550832748413, -0.5450732111930847, 0.2673398554325104, 0.930000901222229, 0.12714989483356476, 0.8824306130409241, -0.6088653802871704, -1.599809169769287, -2.3590004444122314, 1.1245019435882568, -1.3768136501312256, 0.5279644727706909, -1.9754434823989868, 1.140313982963562, 0.20115555822849274, -0.5146373510360718, 0.21458998322486877, -0.7616599202156067, 0.18109233677387238, 0.07732250541448593, -2.7965261936187744, 0.30070701241493225, 0.6350767612457275, 1.029104232788086, 0.11936692893505096, 0.4159526526927948, -1.090774655342102, 0.94503253698349, -1.004326581954956, 1.0778944492340088, 0.15728092193603516, -0.5919223427772522, 0.5706897377967834, -0.6023913025856018, 0.4818652272224426, -0.09840990602970123, 0.01627269573509693, -0.326416015625, -0.6796852350234985, -1.0820573568344116, -1.0228015184402466, -2.125150680541992, -0.18296794593334198, 0.980887234210968, 0.5119860172271729, 0.8774267435073853, 1.1761064529418945, -0.840878427028656, 0.16481339931488037, 1.3465964794158936, 0.23709368705749512, -0.8972561359405518, -0.896626889705658, -1.2470344305038452, -0.15785391628742218, 0.6796070337295532, -0.335457444190979, -0.15443474054336548, -0.22379325330257416, 1.3752409219741821, -0.010990734212100506, 1.0765900611877441, -0.21621449291706085, 0.8183140754699707, 1.4339804649353027, -0.10613767802715302, -0.46177253127098083, 0.15038377046585083, 0.171384796500206, -1.3281841278076172, 0.7922444939613342, -0.8176413178443909, 0.523281455039978, 1.2501579523086548, -0.6161414980888367, -0.3340524733066559, 0.018665684387087822, -1.277414083480835, 1.8426581621170044, 0.06964453309774399, -0.413918673992157, 3.0345399379730225, -0.18035145103931427, 1.0894670486450195, 0.4290104806423187, 0.8894010782241821, -1.4102518558502197, -0.00989961251616478, -1.3101747035980225, -0.6403538584709167, -0.08377761393785477, -1.188555359840393, 0.8265869617462158, 1.1510924100875854, -2.112884759902954, 2.0856685638427734, -0.0021335117053240538, -0.8288800716400146, -0.08308586478233337, 1.0278282165527344, -1.1964561939239502, -0.9060974717140198, 1.5324689149856567, 0.3695157766342163, -0.35516148805618286, 0.2950938045978546, -0.6906360983848572, -0.8831272721290588, -0.03893172740936279, 0.6493974328041077, -2.607621908187866, 0.49822527170181274, -0.8362114429473877, -0.7048896551132202, -0.6951885223388672, -0.005966281518340111, -2.10195255279541, 1.369515299797058, -1.0092136859893799, -1.1285693645477295, 1.3864964246749878, -0.8430033326148987, 0.18079641461372375, -1.4512702226638794, -1.106833815574646, -0.2248595952987671, 0.49570199847221375, -0.033800072968006134, -2.7353451251983643, -0.8501030206680298, 0.7938947081565857, -1.3079410791397095, 2.1759347915649414, -1.1426233053207397, 1.9036431312561035, -0.9666773080825806, -0.8270749449729919, -1.180155873298645, 0.8800049424171448, 0.569420337677002, 0.7701557278633118, 0.06633315235376358, 0.4001689553260803, 2.1590566635131836, -0.47500190138816833, 1.3863637447357178, -0.22429966926574707, -0.6312820315361023, -1.4775383472442627, -0.48460817337036133, -1.6014496088027954, 0.605311393737793, 1.0635071992874146, 0.844919741153717, -0.5785896182060242, 0.2586617171764374, 0.8156817555427551, 0.1580931693315506, -0.8575198650360107, 0.01976631209254265, 0.16795046627521515, 1.1781795024871826, -1.4435962438583374, -0.025651007890701294, 0.32865434885025024, -1.0430896282196045, -0.3172861337661743, -0.6471001505851746, 0.7688794732093811, -0.05636109784245491, -1.1535826921463013, 1.095966100692749, -1.1798548698425293, -1.1561945676803589, 0.9200671911239624, 0.02867269143462181, 1.0765117406845093, -1.5294984579086304, 0.5127735733985901, 1.0490163564682007, -0.3653698265552521, -0.29642945528030396, 0.22348886728286743, 0.19797082245349884, 0.2526669204235077, -0.43755364418029785, 0.060920555144548416, -0.4797894060611725, -1.1047953367233276, -0.9965420961380005, -0.5874759554862976, 0.9642651081085205, -1.104109287261963, 0.4057440161705017, 0.38381198048591614, 1.0704245567321777, -0.3327474296092987, 0.6548931002616882, -0.5063261389732361, 0.7648793458938599, -0.7517394423484802, -0.35330429673194885, 0.24553972482681274, 0.06528054177761078, -0.011123893782496452, 0.3949923515319824, -0.7139787673950195, -0.822277843952179, 2.287780523300171, 0.5521891117095947, -1.292616844177246, -0.8199083209037781, 2.962200880050659, -1.8280235528945923, -0.09512083977460861, -0.09791018813848495, -0.874218225479126, -2.097893476486206, -0.2262388914823532, -0.8443264961242676, -0.4812451899051666, 1.0418025255203247, -0.6940219402313232, -2.7500972747802734, -0.5964644551277161, -1.102096676826477, -0.5819919109344482, 1.5423730611801147, 0.6454167366027832, 0.7764485478401184, -0.7925239205360413, 0.46573176980018616, -0.8380097150802612, -0.5121356844902039, -0.08195099234580994, -0.6014828681945801, -1.4682462215423584, 0.01460738480091095, 0.7126920819282532, 0.5559210777282715, 0.6610899567604065, -1.0362653732299805, 1.0176841020584106, 0.19911009073257446, -1.5079853534698486, -1.247943639755249, 0.3763219714164734, 0.46901172399520874, -0.6657256484031677, -1.063072681427002, 0.8867449760437012, -0.040886975824832916, 0.4510869085788727, 0.6119308471679688, 0.09928445518016815, -1.0561760663986206, -0.011359518393874168, -0.8751180171966553, 1.7286646366119385, -0.4234672486782074, -0.8306469321250916, 0.047113366425037384, -0.8660980463027954, -0.6737627983093262, 1.2787041664123535, 0.46554574370384216, -2.358278274536133, 0.32200074195861816, -0.2079659104347229, -1.5661625862121582, -0.05052695795893669, -0.8866157531738281, -0.34126073122024536, -1.3129706382751465, 0.14996850490570068, -0.9072768688201904, 0.6120113730430603, 2.7985355854034424, -1.6132941246032715, -1.2839605808258057, 0.08290664106607437, -0.4545298218727112, -0.02724681794643402, 0.25822216272354126, 1.6591910123825073, -0.05854421108961105, -1.3491305112838745, 1.3447227478027344, 0.021122636273503304, -0.5092198848724365, -0.14852537214756012, 0.5725958347320557, 1.7977195978164673, -0.6080437898635864, -1.6498216390609741, -3.5272884368896484, -0.05054429545998573, -0.19699425995349884, -0.3553782105445862, 0.19063279032707214, -0.8799887895584106, 1.2305277585983276, -0.664601743221283, -1.2487523555755615, 0.20033878087997437, 1.2000112533569336, -0.46421703696250916, 0.2975340783596039, 0.8516883254051208, 0.0335545688867569, -1.6974098682403564, 0.661007285118103, -1.6504542827606201, 0.8007938861846924, 0.5194726586341858, -0.44008487462997437, 0.9266256093978882, 1.3157215118408203, 1.1186000108718872, -1.638698697090149, 1.6525596380233765, 0.595887303352356, 0.6630590558052063, 2.0119476318359375, -0.7246846556663513, 1.307337760925293, -0.42242589592933655, 0.41275933384895325, 1.0858592987060547, 0.3937923014163971, 1.3024463653564453, 0.5166693925857544, -1.0053163766860962, 0.6680129170417786, 1.3100523948669434, 0.8731892108917236, 1.0645196437835693, 0.9208518266677856, -0.6234053373336792, -1.172390103340149, 2.8912851810455322, -1.418952465057373, 1.6112196445465088, -0.024226846173405647, -0.06716670095920563, 0.25829312205314636, -0.4579628109931946, -2.0401124954223633, 0.06014474481344223, -0.2668830454349518, 0.15132907032966614, 0.17571543157100677, -0.7039997577667236, -0.5857141017913818, 4.047564506530762, 2.233307123184204, 0.23079003393650055, -2.16029691696167, 0.756597101688385, -0.39770832657814026, -1.0635383129119873, 1.1162903308868408, -0.4965566396713257, -0.13891275227069855, 0.3674205243587494, 0.12543527781963348, 0.1870398223400116, 0.01956366002559662, -0.2927321493625641, 1.420483946800232, 0.3052174746990204, -0.0052122874185442924, 0.3106550872325897, 0.44263845682144165, -0.9321659803390503, -0.45228737592697144, -0.4546378254890442, 0.008877976797521114, -2.015132188796997, -0.37359046936035156, 0.37415334582328796, 1.4830232858657837, -0.7291358113288879, 0.08104851096868515, 0.25138550996780396, 0.9457011222839355, 0.14292123913764954, 0.24913489818572998, 0.5736051797866821, -0.45152100920677185, -2.0679032802581787, 1.0121067762374878, 0.3700842261314392, 0.4682924151420593, -0.8179197311401367, 1.1612614393234253, 0.4136659801006317, -0.355901300907135, 0.18858814239501953, -0.5850467085838318, -0.7560837268829346, 0.3827950060367584, -0.3659484088420868, -0.9658454060554504, 0.29613324999809265, -0.39673736691474915, 1.803402066230774, 0.2455083131790161, 0.40493613481521606, -0.9264246225357056, -1.0915764570236206, -0.7745622992515564, -0.24182820320129395, -0.23038309812545776, 2.237943649291992, -0.08704634010791779, -0.05394019931554794, 1.448786735534668, 1.7456430196762085, -1.1699516773223877, 1.0704221725463867, -0.36162376403808594, -1.7797414064407349, 0.9360969066619873, -0.5988590717315674, -1.0252820253372192, -0.9408316612243652, 0.22192998230457306, -1.3156205415725708, -0.1472298800945282, -0.05317885801196098, 0.6982921361923218, 0.33949974179267883, 0.2925974130630493, 0.2835632562637329, -1.2219313383102417, 0.0467773899435997, 0.611789345741272, 0.7448242902755737, -0.041115887463092804, 1.0287755727767944, 1.144257664680481, 0.7258771061897278, -1.6685755252838135, 0.1214882954955101, 0.3685039281845093, 0.9862803816795349, -1.097856044769287, 0.20437519252300262, -1.7633399963378906, 0.5451944470405579, -0.18834447860717773, 0.011311507783830166, 0.17990843951702118, -0.38876107335090637, 0.6949785947799683, -1.7139567136764526, -0.9404328465461731, 0.8017562031745911, -1.5017497539520264, 0.10227523744106293, -1.677282452583313, 0.5839687585830688, -0.6596351861953735, 0.9462348818778992, -0.31624406576156616, -0.14279741048812866, 1.7639415264129639, -0.18559934198856354, -0.16853532195091248, 0.09092032164335251, 3.063438653945923, 0.02663026936352253, 1.9449626207351685, 0.30954408645629883, -0.4255402684211731, -0.7733559012413025, 1.27052903175354, -0.5787250995635986, -0.9828847050666809, -0.7623915076255798, 2.3009676933288574, 1.814373254776001, -1.8357657194137573, -0.2512321174144745, -0.9461919665336609, -0.5214229822158813, -0.027521787211298943, 0.5512310862541199, 0.18277879059314728, 0.5303253531455994, 0.753391683101654, -0.2834940254688263, -0.9900107383728027, 0.3570927679538727, -0.3673778176307678, -0.5879687666893005, -0.6500336527824402, 1.124749779701233, 1.269334316253662, -0.5506986975669861, 1.2943849563598633, -1.8502253293991089, -2.2244060039520264, -0.18724291026592255, -0.6626266241073608, 0.8966837525367737, 0.6841530203819275, 0.2578824460506439, 0.9995194673538208, -0.9835145473480225, -0.17851531505584717, 0.7351550459861755, 0.8554593324661255, -0.5401691198348999, 0.06726278364658356, -0.6790949106216431, 1.585141897201538, -0.255135178565979, 0.7236407399177551, 0.047401729971170425, -0.1768302321434021, 0.36735984683036804, -1.0663151741027832, -0.1053721010684967, -0.34547969698905945, -1.2414660453796387, -1.2026805877685547, 1.5779086351394653, -0.7249142527580261, -1.2954365015029907, -0.9845971465110779, 0.667026162147522, 1.9118287563323975, 0.8314329981803894, -0.9830798506736755, 2.1686339378356934, 2.0104920864105225, 0.2522415220737457, 0.7750794887542725, 0.6766074299812317, -1.7310818433761597, -0.3404683768749237, -1.921305537223816, -0.3981596529483795, -0.5874862670898438, 1.0463588237762451, 0.22057132422924042, 0.3346601128578186, 1.102040410041809, 0.8646301031112671, -0.61624675989151, -0.8193330764770508, -0.7602816224098206, -0.1800951510667801, -1.2200294733047485, 1.4190508127212524, -1.0866978168487549, 0.07073681801557541, -1.0443352460861206, 1.0480332374572754, 0.42737463116645813, -0.9415079355239868, -0.3720853328704834, 1.395078182220459, -1.1269652843475342, -0.10409099608659744, 0.1285465508699417, -0.11971629410982132, -0.08264920860528946, 0.7172646522521973, 1.1097749471664429, 0.6410549879074097, -0.4721875786781311, -2.093773126602173, -0.8431658148765564, -1.5744684934616089, -1.107310175895691, 0.3732133209705353, 0.060494113713502884, 1.4000318050384521, -0.46099868416786194, -0.14436286687850952, 1.2958635091781616, 1.4180165529251099, 0.43651291728019714, 0.66199791431427, 0.4985077977180481, -0.15016597509384155, 1.2718397378921509, 0.08423757553100586, 0.11554404348134995, 0.2770213186740875, 1.08905029296875, 0.42049628496170044, -1.444742202758789, -0.32909801602363586, 0.06390561163425446, -1.0879014730453491, -0.4383794665336609, 1.9480268955230713, 0.9601910710334778, -0.6550547480583191, -0.32067400217056274, -0.2822560667991638, 1.4376972913742065, 0.7706051468849182, 1.4914686679840088, -0.41680672764778137, 0.695405125617981, -0.31960251927375793, -0.9192786812782288, -0.7237423062324524, -1.1952131986618042, 1.316769003868103, 0.22781524062156677, -0.9609168767929077, 0.3775412142276764, -0.5865161418914795, -0.7579854130744934, 0.34522876143455505, -1.4956483840942383, -0.17757797241210938, -0.8425464034080505, 0.11676576733589172, 0.45566797256469727, 0.19253312051296234, 0.8179314136505127, -0.45380258560180664, -0.1264604777097702, 0.07818648964166641, -1.359513282775879, 1.1528434753417969, -0.7167043685913086, -0.37134718894958496, -1.1773911714553833, -0.7356883883476257, 0.8502821326255798, -0.04092881456017494, -1.1965104341506958, -0.14138410985469818, 0.018805453553795815, 0.07515724003314972, 0.5893653631210327, -0.404113233089447, -1.6432782411575317, 0.31212151050567627, -1.3305282592773438, -0.823003351688385, 0.09912173449993134, 0.3985755145549774, -1.193964958190918, -0.038031961768865585, 0.5832474827766418, 0.08454486727714539, -0.05773165076971054, -0.18811476230621338, 0.7316305041313171, 0.039247214794158936, -0.3226310908794403, 0.15874500572681427, -0.21845956146717072, 2.3789689540863037, -0.6919557452201843, -0.9566904306411743, 1.1196209192276, -0.39315569400787354, -0.8500308394432068, 0.37146833539009094, -0.10574175417423248, 0.8390539288520813, 0.21289949119091034, 0.6016151309013367, -0.4986417293548584, -0.7644991278648376, -1.6926013231277466, 1.8237240314483643, -0.960991382598877, 0.4230369031429291, 0.38224250078201294, -1.4751849174499512, -0.4323332607746124, -1.3520199060440063, -0.7780138254165649, -0.2952651083469391, 1.752773404121399, -0.3077864348888397, 0.41446638107299805, 0.9973768591880798, 0.16553525626659393, 0.6330309510231018, -0.12406199425458908, -0.6462258100509644, -2.166299343109131, 0.016568463295698166, -1.8064428567886353, 0.4291001558303833, 0.13077779114246368, 0.7138894200325012, 0.7706871032714844, -0.36209195852279663, 1.2020739316940308, 1.5093402862548828, 0.23239950835704803, 1.4209181070327759, -0.2255629301071167, -0.19302737712860107, 2.232191562652588, -0.7026187181472778, 0.6691921949386597, -0.9509416818618774, 0.15360264480113983, -0.5100945830345154, -0.7679129242897034, 0.010280313901603222, 0.07269462198019028, 1.14997398853302, 0.15135669708251953, -0.13334549963474274, -0.2546102702617645, 0.6714195609092712, 1.4714409112930298, -0.6335507035255432, -0.6721530556678772, 0.222913458943367, -1.0733493566513062, -0.18122722208499908, -2.476638078689575, 0.21231885254383087, -1.4534671306610107, 1.4597312211990356, -1.5064258575439453, 0.39007219672203064, 0.2720833420753479, 1.1094155311584473, 0.8676652908325195, 0.9581279158592224, -0.05319766700267792, 0.33732369542121887, 0.44169679284095764, 0.23279622197151184, 0.9209967255592346, -1.3471119403839111, 0.5188896059989929, 0.408311665058136, 0.15797896683216095, -0.10763515532016754, -0.2714753746986389, -0.28475743532180786, 0.10857195407152176, 0.4900050759315491, 1.3655654191970825, 0.7503975033760071, -2.270127058029175, 0.6072227358818054, -0.7741364240646362, 0.49006175994873047, -0.9036138653755188, 0.5546802282333374, -0.47145307064056396, -0.4516409933567047, 1.045875906944275, -1.2881287336349487, -1.3641870021820068, -1.0109769105911255, 0.8800839781761169, -0.31801798939704895, 0.7019606828689575, 1.0276542901992798, 0.008695348165929317, 1.1168025732040405, -0.5977675318717957, 0.3493275046348572, 1.560271978378296, -1.468581199645996, -0.7289828658103943, -0.544573187828064, -1.309205174446106, 1.708198070526123, -0.4140457808971405, -0.7756370306015015, 0.25742435455322266, 0.06753382086753845, 0.2552877366542816, 0.2835049629211426, 0.6674987077713013, 1.178009033203125], [0.6215243339538574, 1.3924486637115479, -1.9787943363189697, 1.0946074724197388, 1.3886637687683105, -1.227892518043518, 0.2252776324748993, 1.5170613527297974, -0.787632405757904, -0.6659719347953796, 0.34616512060165405, -0.07950176298618317, 0.49366775155067444, 1.1982141733169556, -0.18254373967647552, 1.4956824779510498, -0.3423667252063751, -0.7044211626052856, -0.9696619510650635, -0.018991854041814804, -0.7639428973197937, 1.466766119003296, 1.8567101955413818, -0.15055489540100098, -1.6728228330612183, -1.7014442682266235, 2.4835715293884277, -0.16485580801963806, 0.6140271425247192, -0.9268137812614441, -2.100294351577759, 0.3990252614021301, 0.4675583243370056, 0.05833069607615471, 0.9852614998817444, -0.4177594780921936, 0.2600175440311432, 0.8983022570610046, -1.2220840454101562, 0.3397984802722931, -1.4764621257781982, -1.45204758644104, -0.2772333323955536, -1.8124839067459106, 1.1043766736984253, -0.46366602182388306, -0.9946200847625732, 0.1858982890844345, 1.3820860385894775, 0.021845728158950806, -0.09143143892288208, -0.9279786348342896, 0.2729305624961853, -1.9305671453475952, 1.1701667308807373, -1.0341863632202148, 0.827041745185852, 0.16831009089946747, -0.12166354060173035, 1.6706273555755615, -0.4294794499874115, 0.6134777069091797, -0.6757041811943054, -0.22591891884803772, 0.8891531825065613, -0.40507152676582336, -0.5640767812728882, 0.2736762762069702, 0.8160378336906433, -0.27033862471580505, -1.8019120693206787, 1.06680428981781, -0.200720876455307, -0.8033854961395264, 1.1150158643722534, 0.10850262641906738, 0.1721523553133011, 0.5830963253974915, 1.0333558320999146, 0.9327028393745422, -1.0920875072479248, 0.36871767044067383, 1.0798659324645996, 0.6545668840408325, 1.2188359498977661, 0.6429280042648315, -0.9990182518959045, 0.20191793143749237, -0.020382093265652657, 0.4341278672218323, -0.32185810804367065, 0.023900624364614487, -0.21580097079277039, -0.26862776279449463, 0.044034477323293686, -0.5189807415008545, 0.3533705174922943, 0.03248362988233566, 0.9098758697509766, 1.0043572187423706, -0.9709067940711975, 1.0150865316390991, 0.538094162940979, -1.1185672283172607, -2.9476349353790283, -1.3752237558364868, -0.3633159101009369, 0.7731653451919556, -0.27057015895843506, 2.459317922592163, -1.0748095512390137, 0.2266915887594223, -0.1472325474023819, -0.7230563163757324, -0.013232600875198841, 1.335179328918457, 0.882565438747406, -1.8051657676696777, -0.33266007900238037, 1.727089285850525, 0.4832506775856018, 0.12222132831811905, 0.43155768513679504, 0.4594382345676422, -0.8648006916046143, 2.0445234775543213, -0.9171032905578613, 1.613251805305481, 2.7781946659088135, 1.5654513835906982, -1.1058948040008545, 1.2742040157318115, -0.9017358422279358, -0.32039570808410645, -0.5077468752861023, -0.11509275436401367, -0.7989234328269958, 0.5185205936431885, -0.5089396834373474, -0.00227042636834085, -0.930070161819458, 0.16606831550598145, 1.603091835975647, 1.5720089673995972, -1.262976884841919, 0.49349889159202576, 1.447298288345337, -0.3178906738758087, 0.20652621984481812, 0.7111417651176453, 0.9984139800071716, 0.6543299555778503, 0.307047039270401, -0.6970950365066528, -1.9525086879730225, -0.30336618423461914, 0.7487155199050903, -0.10147512704133987, 1.5515367984771729, -1.087737798690796, 1.304397702217102, 0.2209869772195816, -1.4643120765686035, -0.7926136255264282, -1.0397348403930664, -0.6233303546905518, 0.059024788439273834, 0.6903801560401917, -0.167911559343338, 0.016447599977254868, -0.812858521938324, 1.282520055770874, -1.2010947465896606, 0.19789722561836243, -1.6383148431777954, -0.8602083325386047, -0.09550032764673233, 0.018099840730428696, -0.0013697099639102817, -0.3310783803462982, -0.15576112270355225, 0.8393893241882324, 0.5026625394821167, 1.365789532661438, -1.8842277526855469, -0.5446902513504028, 0.5162436962127686, -1.3368278741836548, -0.46972736716270447, -1.5439403057098389, -1.5473634004592896, 0.5767285227775574, -0.900950014591217, 2.2922539710998535, 0.377530038356781, -0.44461706280708313, 0.9000988006591797, -0.1663442850112915, 0.19859452545642853, 1.6425886154174805, -0.6026456356048584, -0.01990129053592682, 0.6195417046546936, 0.8555693626403809, -2.026761531829834, -0.24606363475322723, 2.824470043182373, 0.30948853492736816, 0.5615678429603577, -0.5545430779457092, 1.0168020725250244, -1.399116039276123, -1.8704485893249512, -0.34573179483413696, -0.49106505513191223, 0.4635268449783325, 0.8914248943328857, 0.06818162649869919, -0.4761907756328583, 0.3552641272544861, -0.4864600598812103, 1.2139641046524048, -0.8736100196838379, -0.4586028754711151, 0.764381468296051, -0.4520186185836792, -0.28230950236320496, 0.47349539399147034, 0.04331950098276138, -1.268417239189148, -0.07746659964323044, 2.2481508255004883, 0.7375694513320923, -1.2533371448516846, 1.0532267093658447, -1.6060175895690918, 0.3669425845146179, -0.14928698539733887, 0.2539431154727936, 0.7669221758842468, -0.08383390307426453, -1.257122278213501, -0.23189987242221832, -0.29190579056739807, 1.71123206615448, -1.983404517173767, -0.8833701610565186, 0.5362508893013, -0.35181865096092224, 0.9759041666984558, 0.6400553584098816, 1.4122462272644043, -1.8192310333251953, 0.8804672360420227, 1.0909359455108643, 0.699496328830719, 0.6940262317657471, 0.4591922461986542, -0.45668846368789673, 0.7322241067886353, -1.139167308807373, -0.4709974229335785, 0.13938772678375244, 0.8347043395042419, 0.38228839635849, 0.1981307417154312, -1.5412869453430176, -0.47351157665252686, 0.6692611575126648, 0.8870244026184082, 0.0009506696951575577, -0.005350242834538221, -1.0412245988845825, 0.9288927912712097, -0.2691021263599396, -0.09055988490581512, -0.8318161368370056, 0.7144933938980103, 1.039215087890625, 0.986029326915741, -1.0256438255310059, -0.4590984582901001, -0.32115164399147034, -0.17785614728927612, 0.41618046164512634, -0.04125555232167244, -2.134546995162964, -0.12255565822124481, 1.3119924068450928, -0.43096137046813965, 3.169686794281006, -0.4649231433868408, -0.30713266134262085, 0.03821583837270737, -0.7168006300926208, -1.5890394449234009, -0.9870451092720032, 2.0857954025268555, -0.33484920859336853, -0.2032543271780014, -1.0001016855239868, -1.1050430536270142, -1.9908604621887207, 0.4165014326572418, 1.069878101348877, 0.36230164766311646, 1.3297356367111206, 0.44384270906448364, 0.5127385258674622, -0.37594786286354065, 0.22935062646865845, -0.28022053837776184, -1.6145602464675903, 0.465210497379303, -2.055487632751465, 0.23115947842597961, 0.6628888249397278, 2.810849189758301, -0.36357563734054565, 0.6313557624816895, -0.10405752807855606, -0.16787919402122498, -0.08624041825532913, 1.1458712816238403, -0.6205416321754456, 0.15998795628547668, -1.4738620519638062, -2.0842273235321045, -1.1956872940063477, 0.8751886487007141, 0.03628751263022423, 0.08867700397968292, -0.27283740043640137, 0.7209055423736572, -0.4224982261657715, 0.6688106656074524, 1.8482919931411743, -0.9484176635742188, 1.3191264867782593, 0.8334514498710632, -1.4959865808486938, -0.9473751187324524, 3.4322237968444824, 1.0947957038879395, 0.6034899353981018, 0.11216451972723007, 0.01787925697863102, 1.5561083555221558, 1.8981226682662964, -1.3397212028503418, 0.2950909733772278, 0.5283078551292419, -0.1576627790927887, 0.5318964719772339, -0.6924396753311157, -0.24990572035312653, 0.773779034614563, -0.5882358551025391, -0.21804866194725037, 0.5072982907295227, 0.14076577126979828, 1.5364766120910645, 0.21607957780361176, -1.000745177268982, 1.051184892654419, -1.600817084312439, -0.9681384563446045, -0.6702826023101807, -0.5070427656173706, -0.6036474108695984, -0.7334781289100647, -0.0073949191719293594, 0.8581472635269165, 1.2596949338912964, -0.20288322865962982, -1.693904995918274, -0.6331983804702759, -0.9162706136703491, 1.8727829456329346, -0.8363705277442932, -2.373490333557129, -1.2732268571853638, 1.5257025957107544, 1.4285404682159424, -0.8156103491783142, 0.8529501557350159, 0.013233431614935398, -1.8169138431549072, 0.40991565585136414, 0.013628751039505005, -0.15231290459632874, -1.6482981443405151, -1.7309095859527588, -1.3595528602600098, -0.2847840189933777, 1.329601764678955, -2.0349159240722656, 0.4601016342639923, -1.5089621543884277, -0.43574613332748413, 0.0630652904510498, -2.9992198944091797, -1.0991525650024414, 0.10398366302251816, 0.581379234790802, 0.43615663051605225, 1.0790616273880005, -0.11854319274425507, 0.15910087525844574, -0.1630442589521408, 0.49050506949424744, 0.6222352385520935, -0.22372916340827942, -0.19452150166034698, -0.27363693714141846, 0.3666124641895294, 1.07108736038208, 0.20666883885860443, 1.9756247997283936, 1.552663803100586, -1.1006804704666138, 1.6310391426086426, -0.0143522247672081, 1.2459707260131836, 0.657580554485321, -0.21456199884414673, 1.045459270477295, 0.016175851225852966, -1.0954452753067017, 1.9831150770187378, 3.1024489402770996, 1.7865526676177979, -0.4109630882740021, 3.3190455436706543, -1.2017755508422852, -1.040366291999817, -1.010846734046936, 0.29704877734184265, -0.3130040466785431, 0.25640633702278137, 0.19572104513645172, -1.6623945236206055, 0.6961290836334229, -0.9115612506866455, -1.1181073188781738, 0.24226874113082886, 0.4782184064388275, 0.8149614334106445, 1.1365196704864502, -0.6618008613586426, -1.051041841506958, 0.7093254923820496, -0.906494677066803, 1.6398110389709473, 0.06747850030660629, -0.25706005096435547, -0.8618377447128296, 0.7493240833282471, -0.467297226190567, -0.1750590056180954, -0.2594706416130066, 0.0007531758164986968, 0.23487752676010132, 1.7436259984970093, 1.2306653261184692, 1.0031377077102661, -0.33242928981781006, -0.9065078496932983, -1.2937134504318237, 0.18828816711902618, 0.5276753306388855, 0.6772560477256775, 0.12093392014503479, 0.36754289269447327, -0.4452507495880127, 0.2028590887784958, -0.5976234078407288, -0.6787951588630676, 0.5603649616241455, -1.4354573488235474, 1.3628414869308472, 1.7685893774032593, -0.9982768893241882, 0.9580954313278198, 0.0909094288945198, -0.3151111602783203, 1.2616397142410278, -1.1563613414764404, -1.1992439031600952, 0.9480016231536865, -0.35376060009002686, 1.8207371234893799, 0.1019158810377121, -0.09948430210351944, -0.6547455787658691, 2.4544832706451416, 0.4410451352596283, 0.08928074687719345, 0.40760013461112976, 0.12375660985708237, 1.2418814897537231, -3.0016331672668457, -1.3982075452804565, 0.49417173862457275, -2.034181594848633, -1.127102017402649, 1.1726038455963135, -0.4228733479976654, 1.6886788606643677, 1.4452623128890991, -0.7439493536949158, -0.2436760812997818, -0.4231351613998413, -0.14011716842651367, 1.3115558624267578, 0.6633478999137878, -0.3287476897239685, -0.3876015841960907, 1.434205174446106, 1.1543936729431152, 0.9289683699607849, 1.0126583576202393, 0.33088546991348267, 1.2840725183486938, -1.4592753648757935, -0.7530646324157715, -0.44026151299476624, -0.27369633316993713, 0.9020771384239197, 0.8760444521903992, 2.2332382202148438, -0.15688921511173248, -0.6197872757911682, -2.1045548915863037, -0.3618689179420471, -0.47138428688049316, 0.7407917380332947, 0.4494878649711609, 2.8741116523742676, -0.21473486721515656, 0.3768618702888489, 0.527632474899292, 1.6233320236206055, -1.228265643119812, -1.9266153573989868, 0.4460241198539734, 1.856679081916809, -1.209458589553833, -1.3429456949234009, -1.058022141456604, 0.18430589139461517, -0.933709979057312, 0.26140865683555603, 1.6217666864395142, 0.8760867714881897, 0.43800362944602966, -0.4907335042953491, 0.23822243511676788, 2.75203013420105, -1.1159027814865112, -1.1306246519088745, -0.9641686677932739, -0.7824159264564514, -2.1013309955596924, -1.0797004699707031, -1.3842134475708008, 0.789925217628479, 1.7518656253814697, -0.8789239525794983, -0.6507415175437927, -0.3905060291290283, 0.24422092735767365, -0.9574887752532959, 0.8284652829170227, -0.7005683779716492, 0.5355508327484131, -2.3185524940490723, -0.10476260632276535, -0.6365503072738647, 1.0995243787765503, 0.1912156045436859, 1.166141152381897, 0.42376914620399475, -0.5817333459854126, -1.0700803995132446, -0.1447242647409439, 1.1285851001739502, -0.6675590872764587, -1.194926381111145, 0.8169847726821899, -0.0344671793282032, -0.5176384449005127, -1.0210731029510498, -0.2548541724681854, 0.5151473879814148, 0.8575378060340881, -0.4288846552371979, -1.1465548276901245, -0.5482346415519714, 1.2417793273925781, 0.43960654735565186, 0.8443366885185242, -0.006684676744043827, -0.0843016728758812, -0.761713445186615, -1.2434749603271484, -0.4106968343257904, 0.3376409709453583, 0.007505482062697411, 0.8041301369667053, -0.1937064528465271, 0.09523716568946838, -1.8762444257736206, -0.05216464772820473, -0.55946946144104, -0.8902215361595154, 0.4086153507232666, -0.09268341213464737, 1.9108870029449463, -0.05583340302109718, 0.745389997959137, 1.456355333328247, 2.0686147212982178, -1.469977617263794, -0.7288094758987427, 0.005115557461977005, 0.29110315442085266, 2.193549871444702, 2.09812593460083, -0.6806699633598328, -0.24942976236343384, -0.2729238271713257, 1.3190195560455322, -0.05413307994604111, 0.36892667412757874, 0.2457055151462555, -1.1417416334152222, -0.37279003858566284, 0.03461109474301338, 0.21580442786216736, -0.10855628550052643, -0.08076263219118118, -0.0009462194866500795, 0.5413954257965088, -0.036748215556144714, 0.1969565898180008, -0.5583395957946777, -0.6716533899307251, -1.29859459400177, 0.36690834164619446, -0.07987857609987259, 0.5665761232376099, 0.09729965776205063, -1.0626972913742065, 0.303961843252182, 0.37560540437698364, 0.807256817817688, -1.3773078918457031, 1.1534887552261353, -1.2888132333755493, 2.492969512939453, 0.8077526092529297, 0.5559787154197693, 2.880784034729004, -0.26863324642181396, -0.7789799571037292, -0.7117248177528381, -0.39567968249320984, -1.5713372230529785, -0.6956496834754944, 2.470794916152954, -0.7834568619728088, -1.0856012105941772, 0.8910120725631714, 0.627346396446228, -0.12833614647388458, -1.5309149026870728, 1.0783860683441162, -0.6892011165618896, -0.5029672980308533, 0.3167050778865814, 1.591770887374878, -0.1392708122730255, -0.5789676308631897, 1.170792579650879, 1.9664274454116821, -1.4181617498397827, 0.22243726253509521, 0.22018666565418243, -0.061537545174360275, 0.728158175945282, -1.111538052558899, -0.18625040352344513, -0.6487798094749451, -0.6711388230323792, 0.28597772121429443, 0.16585984826087952, -0.12263217568397522, 0.6603330969810486, -0.1934237778186798, -1.8403863906860352, 0.5973621606826782, 0.8556662201881409, 1.2504403591156006, -2.821119785308838, -0.029706863686442375, -0.3599949777126312, 2.1413426399230957, -0.9060791730880737, -0.10396023839712143, -0.17980808019638062, -0.38115668296813965, 1.0698009729385376, 0.4451652765274048, -0.5089783668518066, 0.8166556358337402, 1.768088459968567, 0.6509443521499634, 0.3392617702484131, -0.7380011081695557, 0.5792359709739685, -0.16360792517662048, 0.44727814197540283, 1.324609398841858, 0.3874839246273041, 3.0316379070281982, -1.2993016242980957, 0.15816114842891693, -0.8963380455970764, 1.2459434270858765, -1.6149042844772339, -0.42508089542388916, -1.4203146696090698, -2.1543641090393066, 0.43524661660194397, 1.4432432651519775, 0.8793227672576904, -1.1019487380981445, -2.0301859378814697, -0.6796025633811951, -0.009246173314750195, 0.467104971408844, -1.1544044017791748, -0.8075655698776245, -1.0414385795593262, 0.90452641248703, -0.5141863822937012, -0.35817810893058777, 1.2947065830230713, -0.6226508617401123, 0.3425024747848511, 0.4696679413318634, 1.7015652656555176, 0.9946644902229309, 0.6175032258033752, 0.9160863757133484, -0.35604336857795715, 0.8718900084495544, -1.5111429691314697, -0.39494866132736206, -0.31604936718940735, 0.525046169757843, 0.10449334233999252, -0.060013044625520706, 0.18136334419250488, -1.6429438591003418, -1.4254302978515625, -0.08144568651914597, 0.5589480996131897, 0.39336636662483215, -1.232366681098938, 2.6150217056274414, -0.09157786518335342, 0.596674382686615, 1.1006649732589722, 0.43514567613601685, 0.2957299053668976, -0.895487368106842, -0.586406946182251, -1.1351855993270874, -0.656029999256134, 1.1671478748321533, 1.1426031589508057, 0.49004197120666504, -0.5498402714729309, -2.349968910217285, 0.5668259263038635, -0.8306475877761841, -0.7152734994888306, -0.6326594352722168, 2.6505823135375977, -1.756674885749817, -0.41871175169944763, -0.7158416509628296, -0.4347246289253235, 0.8271636366844177, 0.030799025669693947, 0.7954278588294983, 1.3137584924697876, -0.3751527667045593, 0.8108506798744202, 0.0849449560046196, 0.9798535108566284, 1.3685400485992432, 0.9653968811035156, 1.0253852605819702, 0.4170680642127991, 0.11321073770523071, 0.5230615735054016, -0.0996246188879013, 0.10450411587953568, 0.6677576899528503, -0.393258661031723, 0.3531735837459564, -0.2707321047782898, -0.500083327293396, -1.0521574020385742, -0.7716591358184814, -0.11281011253595352, 0.5418028235435486, 0.6274507641792297, 0.7974483370780945, -1.6291533708572388, 0.06823877990245819, 0.8244554400444031, -1.3455309867858887, -1.6987733840942383, 0.8270140886306763, -1.064987063407898, -0.0752699077129364, 0.12878185510635376, 1.4178977012634277, -1.3084726333618164, -2.1717963218688965, 0.5718168616294861, 1.2067475318908691, -1.0640145540237427, 0.05344293639063835, 0.789307177066803, -1.2960587739944458, -1.3967480659484863, -1.3616201877593994, -0.19945304095745087, -1.7127913236618042, -1.1915366649627686, -0.205025777220726, -0.08728619664907455, -0.5298953056335449, 0.04633982107043266, 1.5710642337799072, -1.0533404350280762, -1.0555421113967896, -0.7972180247306824, 0.4194623529911041, 0.9199142456054688, 0.05838286131620407, 0.8651815056800842, -1.3005257844924927, -0.491828590631485, -1.7267225980758667, 1.366401195526123, -2.5455310344696045, 1.4179421663284302, 1.7386947870254517, -0.5973795652389526, -0.1805540770292282, -0.3752335011959076, -0.4458146393299103, -0.7459316253662109, 1.5959625244140625, -1.154726266860962, -1.0335302352905273, 0.41830793023109436, 0.019427461549639702, 0.738926112651825, -0.995604395866394, 0.6824792623519897, 0.5524784326553345, 0.17695598304271698, -0.7166327238082886, -0.5072515606880188, -1.2408462762832642, -0.07848457992076874, -1.3000987768173218, 0.5061351656913757, 1.950581669807434, 1.4146933555603027, -2.5578248500823975, -1.515453577041626, 1.368276596069336, -1.7952749729156494, 1.6607308387756348, -1.5706053972244263, 0.42606499791145325, -2.128706455230713, 0.2331348955631256, 0.3940922021865845, 0.09900076687335968, 0.1379757970571518, 0.351393461227417, -2.1915838718414307, 0.5966185331344604, -0.7580385804176331, -0.46279406547546387, -0.7838999032974243, -1.7523037195205688, 0.6662120223045349, 0.4161430597305298, -0.7218835353851318, -0.6108272075653076, -0.526585042476654, -0.733824610710144, -2.027078628540039, 0.19396105408668518, 0.1482628583908081, 0.6219987273216248, -0.7361665368080139, -0.7294893264770508, -0.45040780305862427, -1.2292567491531372, -0.07431304454803467, -0.006947229150682688, -1.246150255203247, 1.8760920763015747, -1.983099102973938, 0.04351408779621124, -0.2370154857635498, 0.676528811454773, -0.9238205552101135, 0.611402690410614, 1.162149429321289, -1.0397169589996338, -1.3283178806304932, -1.4835987091064453, 0.5088309049606323, -0.9696289896965027, 0.25114765763282776, 0.22504325211048126, -0.26259711384773254, 1.3527816534042358, 0.39086857438087463, 0.35161247849464417, 1.4406334161758423, -1.7834187746047974, 0.5967758297920227, -0.9434822797775269, -0.24632982909679413, -0.579559862613678, 0.22789925336837769, -0.8011900782585144, -0.7653326392173767, 1.7477688789367676, 0.850400447845459, -0.6550844311714172, -0.5676022171974182, -1.1736830472946167, -0.49532899260520935, -0.39239001274108887, -0.10344963520765305, 0.5695932507514954, 0.7163209915161133, 0.7571430206298828, -0.0025273903738707304, 1.1545321941375732, 0.7785204648971558, -0.43965592980384827, 0.10327090322971344, 0.03660137951374054, -0.1162155494093895, 0.08219942450523376, -0.36571165919303894, 0.6952229738235474, 0.6854976415634155, -0.45773565769195557, 0.7001604437828064, -1.530181884765625, -0.3504956066608429, -0.6253897547721863, -0.36971360445022583, -0.1532810479402542, -1.1333750486373901, -0.8050954937934875, -0.17367732524871826, 0.8126956224441528, -0.12402024120092392, 0.9753212928771973, -0.7149859666824341, -0.419151246547699, -0.35641372203826904, -1.0037436485290527, 0.017125511541962624, -1.8044310808181763, -0.9786412715911865, -0.05167314037680626, -0.898595929145813, -0.9832773208618164, -0.03585025668144226, 1.5875134468078613, -0.3270970284938812, 0.2983873784542084], [-0.42409950494766235, 0.9112748503684998, -0.24035051465034485, -1.0480774641036987, -1.3200585842132568, 0.32341477274894714, 0.3403131067752838, 1.7926572561264038, 0.04131317511200905, -2.876000165939331, 0.6750004887580872, 0.5824972987174988, 1.1864995956420898, -2.232525587081909, 0.013989996165037155, -0.09588328003883362, -1.8314285278320312, 0.4860268235206604, 0.5188152194023132, -1.8815518617630005, -0.18268650770187378, 1.05425226688385, -1.8041777610778809, 0.11938437819480896, 0.5277643203735352, -1.389515995979309, -1.2530063390731812, 1.4556127786636353, -0.7277087569236755, 1.4061813354492188, -1.082331895828247, 0.26828962564468384, 1.362962007522583, 1.6088918447494507, 1.7270385026931763, -1.3258506059646606, -1.4556952714920044, 0.9828721284866333, -1.3165082931518555, 0.5240663886070251, -2.6386470794677734, -0.017131980508565903, 1.3207306861877441, -1.5736525058746338, 0.07573281973600388, 0.8275662660598755, 0.25798508524894714, -0.5566108226776123, -1.049161434173584, -0.40060779452323914, 2.6825735569000244, 0.7686036229133606, -1.5944122076034546, -0.4553040862083435, 0.35202983021736145, 0.7103385329246521, -0.5837196111679077, -0.3419342041015625, 0.5172390937805176, -1.172540545463562, -0.47375160455703735, -2.56097149848938, 0.5239646434783936, 1.1484836339950562, 0.22444167733192444, 1.1220680475234985, 1.170911431312561, 0.3493940234184265, -1.9453282356262207, 0.43729639053344727, -1.6267658472061157, 0.3457840383052826, -2.5759174823760986, 1.0805968046188354, -0.3147328495979309, -0.19972339272499084, 1.5222116708755493, -0.6979789733886719, -1.063349962234497, -1.0487984418869019, 0.5918154716491699, -0.045887600630521774, -1.58688223361969, 0.27526944875717163, -0.45621347427368164, 0.5329419374465942, -0.09129668027162552, 0.5501554012298584, 0.6281365156173706, 0.4231334626674652, 0.49614232778549194, -0.8427200317382812, -0.9458038806915283, -1.1624486446380615, 0.836723268032074, 0.5487170219421387, -0.03664533421397209, -1.0650129318237305, 0.9420905113220215, -0.7944536209106445, 0.6550170183181763, -0.3367443084716797, 0.9964096546173096, -1.4195713996887207, -0.2504633665084839, -0.42915937304496765, 0.19979169964790344, -1.303023338317871, 1.1277732849121094, -0.7139943838119507, -0.8535906076431274, 0.25354883074760437, -0.9771351218223572, 0.20661193132400513, 1.836251974105835, 0.1977631151676178, -0.5745315551757812, 1.4758447408676147, -0.7344194650650024, -0.25139856338500977, -0.019197294488549232, 0.2695297598838806, -0.06732653826475143, 0.8101526498794556, 0.6377617120742798, -0.19999821484088898, -0.1920957714319229, -0.7236831188201904, 0.04013514518737793, 0.41345497965812683, 0.6095139384269714, -0.130273699760437, 1.648730754852295, 0.7264653444290161, 0.1924930363893509, 1.4197139739990234, -0.32905781269073486, -2.014446258544922, 1.009467601776123, 0.38202762603759766, -0.36479452252388, -1.8388053178787231, 0.06308668106794357, 0.45783233642578125, 1.6277097463607788, 0.1412978321313858, 1.822653889656067, -1.49192214012146, 0.1064494401216507, 0.015184406191110611, -0.5109140276908875, -0.41227734088897705, 0.718435525894165, -0.41044166684150696, -0.6582322716712952, -1.0046111345291138, 1.24591064453125, 0.38245007395744324, -0.16144472360610962, 1.7787007093429565, -0.5674207210540771, -0.606349527835846, -0.228268563747406, 0.7691595554351807, 0.6446149349212646, 0.6305066347122192, 0.6917622685432434, 1.1193344593048096, -0.4091496169567108, -0.4055907130241394, 2.921449661254883, 1.9839210510253906, -0.2522972822189331, 0.26653286814689636, -0.2408696413040161, -0.9975391626358032, -1.5319042205810547, 0.4848743677139282, 0.23587211966514587, -0.288230299949646, 1.0667368173599243, -0.07127737998962402, 1.2281579971313477, 1.0946742296218872, 0.1776188164949417, 0.31259873509407043, 1.15884268283844, -0.24246181547641754, -0.7130573987960815, -0.15850165486335754, 0.032821524888277054, 0.6202718615531921, -0.8036181926727295, 0.24495887756347656, -0.1507488489151001, 0.016365721821784973, 0.39894136786460876, -0.7800223231315613, -0.9934207201004028, -0.6912292838096619, 1.4010400772094727, 0.5863381028175354, -1.541266679763794, 0.5301113128662109, -2.3568994998931885, 1.0556236505508423, 1.488297939300537, -1.641528606414795, -1.7671875953674316, -0.467810720205307, 0.2821534276008606, 0.9313111305236816, 0.41377338767051697, -1.3174443244934082, 0.7630812525749207, 0.0023404208477586508, 0.21099716424942017, 0.1545848846435547, 0.5241634845733643, 0.039202552288770676, -0.9773630499839783, 1.0884802341461182, -0.5199495553970337, 0.997312068939209, 0.9718033671379089, -0.021860873326659203, -0.2859908640384674, 1.0554313659667969, 0.06899808347225189, -0.2815600335597992, 0.5242890119552612, 0.06455928832292557, 0.5138633847236633, -0.7923519015312195, -0.39274275302886963, 0.5075989961624146, -0.492349237203598, -1.1072731018066406, -0.31218594312667847, 1.110296607017517, 0.9505500197410583, -1.7465752363204956, 0.6720084547996521, -0.22461539506912231, -1.4080283641815186, 0.8464686870574951, 0.8055771589279175, -0.7299701571464539, -1.1749650239944458, -0.5960783958435059, 0.3501027524471283, -0.2773362100124359, 0.6791374087333679, -1.3470699787139893, 1.0693856477737427, -0.48383569717407227, -1.4897189140319824, 1.0395697355270386, 0.33828872442245483, -0.8171566724777222, 0.6431746482849121, -0.019629551097750664, 0.10972920060157776, -0.16111122071743011, -0.7503872513771057, 0.6304987072944641, -1.3392844200134277, -1.3197275400161743, 0.38308683037757874, 0.23100920021533966, 0.2717037796974182, -0.5430411696434021, 0.8779852986335754, 0.45536381006240845, 0.6246317625045776, -1.0765371322631836, 1.3460822105407715, 0.07351867109537125, 0.2162720113992691, 0.10028476268053055, -0.5456292629241943, -0.6655488610267639, 0.7327479124069214, -1.8470947742462158, -0.8740220069885254, -1.3100645542144775, -0.7899150252342224, 0.1244041919708252, 0.2850746214389801, 2.8866240978240967, 0.4579284191131592, -0.4448823630809784, 0.5713809728622437, 0.11552391201257706, -1.2526780366897583, 0.2934942841529846, -1.274111032485962, -0.3970813453197479, -0.7862204909324646, 0.7862903475761414, -0.7275874614715576, 0.20803241431713104, -0.7609611749649048, -0.32034748792648315, -0.3000701069831848, -0.19797946512699127, 0.9135289788246155, -1.0528786182403564, 0.15689852833747864, 0.43636730313301086, -0.921928346157074, 0.8440299034118652, -1.5532814264297485, -0.2967170476913452, 0.9436676502227783, 1.211938500404358, -0.9957152605056763, 0.5792088508605957, 1.3876549005508423, 0.20786643028259277, -0.36465638875961304, 1.2745425701141357, -1.4471797943115234, -0.36571598052978516, 0.6274393796920776, 1.376595377922058, -0.3668813407421112, 0.9917902946472168, 3.3520729541778564, 1.3124908208847046, -0.2908317744731903, -0.3496265709400177, -0.5051684379577637, -2.0994226932525635, -0.906165599822998, -1.04874849319458, 0.15912024676799774, -0.9315805435180664, 1.4021220207214355, 0.07260351628065109, 0.4633122384548187, 0.125223308801651, 0.9854162335395813, 0.8965543508529663, 1.1376060247421265, -1.1062051057815552, -0.2901296615600586, -0.635960578918457, -0.2891095280647278, 0.6749115586280823, -1.812359094619751, -0.5100512504577637, -1.3802236318588257, 1.3873751163482666, -1.1560908555984497, 1.0737687349319458, -0.7459347248077393, -0.6453000903129578, 0.8636630177497864, 1.4614951610565186, 0.7494768500328064, -1.1921117305755615, 1.0850727558135986, -2.9900808334350586, 1.6790459156036377, -1.317328691482544, -0.2952631413936615, 1.8549044132232666, 0.1815948784351349, 1.0826337337493896, -0.7171085476875305, 0.2369697391986847, -0.1600334346294403, 0.06791865080595016, 2.467381477355957, 0.33614304661750793, 0.06758394092321396, -1.3710874319076538, -1.1899032592773438, 0.39975228905677795, 0.42388197779655457, -0.7109414339065552, -0.7062776684761047, 1.4813101291656494, 0.16292604804039001, 0.39454221725463867, -0.7805066704750061, 0.15330399572849274, 1.4410747289657593, -0.9823371767997742, -0.8379751443862915, 0.6542631983757019, -0.586054801940918, -1.4718254804611206, -0.152797132730484, 0.5556654334068298, 0.11839120835065842, -1.9318732023239136, -0.10229344666004181, 0.38678017258644104, 0.8059134483337402, 0.2817672789096832, 1.0439049005508423, 0.08006345480680466, -1.7772694826126099, 0.41918811202049255, 0.2748056948184967, -1.4884711503982544, -1.0332262516021729, 0.9990729093551636, -1.2467397451400757, 0.8155228495597839, 0.43629729747772217, -0.9304782748222351, -0.1453304886817932, -2.09086275100708, 0.3575267493724823, -0.37703725695610046, 0.07987265288829803, 0.08381644636392593, 0.5477989912033081, -0.23676811158657074, -0.40521663427352905, 0.20642414689064026, 0.7300269603729248, -0.9418250918388367, 2.818389415740967, -0.888157844543457, 0.42759233713150024, 1.9027892351150513, -0.8600921034812927, 0.26478442549705505, 0.005980791058391333, -0.5053178668022156, -0.26347029209136963, -0.8607316613197327, 0.9922856688499451, 0.24137794971466064, 1.0582560300827026, -2.1024084091186523, 0.45091956853866577, 0.3070860803127289, 0.34945806860923767, -0.22823792695999146, -1.1758036613464355, 1.205483317375183, -0.15404675900936127, 1.4523848295211792, 0.44088026881217957, 0.32011398673057556, -0.5307582020759583, 0.39642900228500366, -0.3144562840461731, -0.2878602147102356, -0.0564994178712368, 1.5848199129104614, -0.8810497522354126, -0.2540842592716217, -1.1764092445373535, -0.11848187446594238, 0.3313805162906647, 1.593677043914795, -0.11946288496255875, 0.38737285137176514, -0.4469341039657593, 0.5132941007614136, -1.712432861328125, -1.7891050577163696, 0.5344101786613464, -0.750983476638794, -0.004856439772993326, -0.41206929087638855, -0.25873810052871704, -0.6351446509361267, 0.08633850514888763, -1.0430896282196045, -0.9412097334861755, -0.3775656223297119, -0.10392171144485474, -1.3904770612716675, 1.3680986166000366, -2.341078042984009, 1.7586045265197754, -1.9192397594451904, 0.6774217486381531, 0.19632700085639954, 0.08579932153224945, -0.6105626821517944, -2.0240774154663086, -1.6145190000534058, 0.3213062882423401, -0.5548524856567383, 0.552926242351532, -0.14936043322086334, 0.1387217789888382, -0.2889552414417267, -0.5211494565010071, 0.06188211590051651, 0.6642279028892517, 1.0248395204544067, 0.9925336241722107, 1.893505573272705, -0.553348958492279, -0.009090535342693329, 1.338422417640686, 0.0040262253023684025, -2.235971212387085, -1.2565382719039917, 0.38567912578582764, 0.05163291469216347, 0.951738178730011, -1.6506670713424683, 0.5349612832069397, -0.9205651879310608, -0.25454482436180115, -0.4775831699371338, -1.2559581995010376, -0.3957628011703491, -0.8934124112129211, -0.1346907764673233, 0.7239595055580139, -1.6457819938659668, -0.30806592106819153, -0.25967937707901, 0.7109703421592712, -0.32778850197792053, -0.14558030664920807, 1.9374465942382812, -1.372333288192749, -0.317516565322876, 0.18211817741394043, 1.4567402601242065, 2.13747239112854, 0.3740543723106384, -0.05646812915802002, -1.0651618242263794, 1.479706883430481, -1.5137338638305664, -0.2820209264755249, 0.5761077404022217, 0.6411350965499878, 0.21993455290794373, -0.1439371407032013, -0.23397807776927948, 2.4166276454925537, 0.9265230298042297, -1.1873981952667236, 2.2657387256622314, 0.10509765148162842, 0.2479097992181778, -0.5578349232673645, 0.6065133810043335, -0.0014259765157476068, 0.6512293815612793, 1.5220452547073364, 0.37113237380981445, -0.08892972767353058, 0.09702286124229431, 0.4079152047634125, 0.5655832290649414, -1.013655424118042, -0.9696981906890869, 1.1276451349258423, -0.4099895656108856, 1.3400064706802368, 0.031890612095594406, 0.396430641412735, -2.4350972175598145, 0.5623716711997986, 0.08023232966661453, -1.6334937810897827, -0.46446365118026733, -0.06349261850118637, -2.201711654663086, 0.6857027411460876, 0.5296530723571777, 1.0753343105316162, -1.3946237564086914, -0.8315652012825012, 0.8701246976852417, -0.46825456619262695, 0.1040111854672432, -0.0387176051735878, -0.47519975900650024, 0.02177257090806961, 0.23237496614456177, 1.0090289115905762, -0.23607884347438812, -0.077478788793087, 0.4394077658653259, 0.07177513092756271, 0.22994321584701538, -1.5590717792510986, -0.6087560057640076, -0.27592596411705017, 1.3556835651397705, 0.9726822972297668, -1.8085196018218994, 0.3842649459838867, 2.367888927459717, -2.7173829078674316, 0.435199499130249, -0.5647966265678406, -0.19673143327236176, 0.8834654092788696, -0.26691845059394836, -0.765690267086029, -1.9779434204101562, 0.9035351276397705, 0.541449248790741, 0.8234460353851318, -1.1502134799957275, -0.4385693669319153, 0.46323609352111816, 0.4233143627643585, 0.9932351112365723, 0.6978775858879089, 0.27009692788124084, -0.16123858094215393, 0.037283699959516525, 0.19248685240745544, -0.3899976313114166, -0.5553169846534729, -0.21192434430122375, 0.20765496790409088, -0.6719788908958435, 0.06725763529539108, 0.08905311673879623, -0.11755839735269547, 1.3020750284194946, 0.38781461119651794, 0.12125086784362793, -0.1642766296863556, -0.055320415645837784, -1.197074294090271, -0.8656373620033264, -0.3537174165248871, -0.5772370100021362, -0.2028873711824417, 1.2115912437438965, -0.4509323537349701, -0.3329710364341736, 1.4467320442199707, 1.4420192241668701, 1.5683739185333252, -0.8822126984596252, -1.704269289970398, 1.479353904724121, -1.7379547357559204, 2.8222758769989014, -2.550657033920288, -1.7532074451446533, -1.0053340196609497, 0.6231915950775146, 0.8190558552742004, -1.3729954957962036, -0.6555057168006897, 0.7397281527519226, 0.6059949994087219, 0.48307541012763977, -0.23297850787639618, -0.8492088913917542, -0.6987347602844238, 1.1991183757781982, 0.8918594121932983, 0.15511222183704376, -1.0932095050811768, -0.65718674659729, 2.051060199737549, 1.4532833099365234, -0.9521139860153198, -0.6804218292236328, 1.7635599374771118, -1.8278660774230957, -1.428462028503418, -2.0475664138793945, -1.1136882305145264, -0.2607603669166565, -0.133174866437912, 0.9702303409576416, -2.4330172538757324, 0.5510623455047607, 0.4528791308403015, -1.2057371139526367, -0.043933313339948654, -0.6370023488998413, -0.11356634646654129, 0.23516784608364105, 0.9623160362243652, 0.4063096046447754, -0.18499942123889923, 0.036539141088724136, 1.1121151447296143, -0.1790282279253006, -1.7865090370178223, -2.041635751724243, 1.4211212396621704, -0.44826266169548035, -0.42220258712768555, -0.25085484981536865, 0.9174919724464417, 1.1767393350601196, 1.9481498003005981, 0.7153943777084351, 1.1331381797790527, 1.9739091396331787, -0.13507628440856934, 1.6794884204864502, -1.591933250427246, -0.3655098080635071, -0.3542385399341583, 2.292008638381958, -0.03772522136569023, 0.8406347036361694, -0.5520071387290955, 0.25524672865867615, -1.340172529220581, -1.1919476985931396, 0.4552643895149231, -0.9633042216300964, -1.1760194301605225, 0.6879356503486633, 0.4175906181335449, -0.7754427790641785, -0.12656211853027344, 0.31756505370140076, 0.9582782983779907, -0.8297526240348816, -0.8274447917938232, -1.302643060684204, -0.9451298713684082, 0.4311142861843109, 1.6043360233306885, 0.4953831434249878, 1.8672034740447998, 0.5486314296722412, 0.9665123820304871, -0.40990978479385376, 1.0889264345169067, 1.1748062372207642, 0.5979321002960205, 1.5027810335159302, 0.1307472139596939, 0.041155144572257996, 0.9570459723472595, -0.7677992582321167, 0.4935548007488251, 1.4524147510528564, -0.3780750632286072, 1.2038155794143677, 0.39049965143203735, -0.6523054242134094, 1.1368898153305054, 0.6506565809249878, -1.9349206686019897, 0.07319415360689163, 0.1932242065668106, -0.0840754508972168, 0.9025566577911377, 1.2726283073425293, -0.4134303629398346, -0.2309800386428833, 0.9935070276260376, 0.2031031996011734, -0.14662477374076843, 0.07219626009464264, 1.5162861347198486, 0.2869533598423004, 0.456739217042923, -0.2857428789138794, 0.9701363444328308, 1.6559361219406128, 0.520432710647583, -0.33149856328964233, -0.6128054261207581, -0.4184562861919403, -0.5988002419471741, -1.9382286071777344, 1.6316770315170288, 1.5390410423278809, -0.36870571970939636, -0.16958428919315338, 0.6083070635795593, -0.21269038319587708, -0.02263224869966507, 1.3609919548034668, -0.4514337182044983, 1.3049376010894775, 1.3947677612304688, 1.371054768562317, -1.450311303138733, 1.2993948459625244, -0.0008881097892299294, 0.19526399672031403, 0.3944016993045807, 0.9426679611206055, 0.35074156522750854, 0.6548299789428711, 1.4283504486083984, -1.810116171836853, -0.3075316846370697, -0.206129252910614, -0.7212924361228943, -0.07494401931762695, -1.456676959991455, 0.8968992829322815, -0.9303575158119202, 0.15254133939743042, -1.2616382837295532, 0.02958924137055874, 0.027441391721367836, -1.4147485494613647, 0.24369694292545319, -0.3798592984676361, 0.5758715271949768, 0.16832324862480164, -1.0382575988769531, -1.3900564908981323, 0.7749425172805786, -0.5200327038764954, -2.162717342376709, 0.025354500859975815, -0.2417140156030655, 0.7130282521247864, 1.2915350198745728, 0.14638866484165192, 0.9474228024482727, 1.596754789352417, 0.954531192779541, 0.5830525755882263, -0.5609753727912903, 0.8691574335098267, -0.5099494457244873, 0.9717503786087036, 0.8869935274124146, 0.08737655729055405, -1.7844223976135254, 0.8155563473701477, -1.3670334815979004, -0.35201597213745117, -1.976427674293518, -0.45233315229415894, 0.3377052843570709, -0.1847103387117386, 1.7780296802520752, 0.37715476751327515, -1.38797926902771, 0.749372124671936, 0.14688684046268463, -0.5373761653900146, -1.8242007493972778, -0.8694595694541931, -3.0013413429260254, -0.24683231115341187, 0.6612922549247742, 0.6795225739479065, -0.46053770184516907, 0.6451473236083984, 0.06886768341064453, -1.3031704425811768, -1.354090929031372, -0.3049411177635193, 0.0849262923002243, 0.06001545861363411, -0.11181001365184784, -0.41634401679039, 1.3680325746536255, -0.2741132974624634, -1.0366740226745605, -2.0491724014282227, 0.1532684713602066, 1.0218011140823364, 0.7535694241523743, 0.3453407287597656, 1.009118676185608, -0.530778169631958, -0.8486992716789246, -0.886721134185791, -1.4206879138946533, -1.5511466264724731, -1.3021949529647827, -0.29895371198654175, -0.6712692379951477, -0.3774361312389374, -2.965775966644287, 0.18531188368797302, 1.4131510257720947, 1.1961599588394165, -0.27438217401504517, 0.48434221744537354, 2.15195894241333, 0.609045147895813, 0.648344099521637, 0.045656830072402954, 0.9061041474342346, 1.304729700088501, 0.30075982213020325, 0.5239976644515991, 0.9079311490058899, -0.5083284378051758, 0.6793412566184998, 0.7136929631233215, 1.2362079620361328, 3.3077056407928467, -0.6514065265655518, 0.8469934463500977, 0.582444429397583, 1.3576115369796753, 1.0781965255737305, -0.8619643449783325, -0.9961042404174805, -0.5118780136108398, -1.1420313119888306, -0.7827579379081726, 1.0801973342895508, -3.032806396484375, -0.2994431257247925, -1.2914385795593262, -0.35307204723358154, -0.7037922739982605, -1.2382214069366455, -1.0286287069320679, -1.3901304006576538, -1.1591923236846924, -0.9361793398857117, 0.16981245577335358, 0.24485304951667786, 1.7879912853240967, 0.3005129098892212, -0.6784564256668091, 0.37791430950164795, 0.7029536366462708, 1.1614540815353394, 1.1155632734298706, -0.6301037669181824, 0.13548053801059723, 0.5088028311729431, -0.7002145051956177, -1.774689793586731, 0.9050824046134949, 2.239408254623413, 0.8372849822044373, -0.08340354263782501, -1.9057620763778687, 0.4090244770050049, 2.430864095687866, -1.7446973323822021, -0.7165820598602295, 0.17121481895446777, 0.9475011825561523, -0.4559105634689331, -1.6028320789337158, 0.04124264046549797, -0.12576527893543243, -0.9128722548484802, 0.6271489858627319, 0.30349186062812805, 0.19347676634788513, 0.1415066421031952, -0.5861788988113403, -0.6906630992889404, -1.455941915512085, 1.2329933643341064, 0.1290520578622818, 0.34141960740089417, -0.5172112584114075, -0.550717294216156, -1.7062865495681763, 0.13489021360874176, -0.45423388481140137, -0.3444328308105469, -0.23861205577850342, -1.6098344326019287, 1.3042263984680176, 1.823262333869934, -0.6550467610359192, 0.3147510588169098, 0.14531812071800232, 1.134138822555542, -0.16981995105743408, -0.8294368386268616, 0.6145541071891785, 0.39312055706977844, -2.230929374694824, 0.4554296135902405, 1.7816565036773682, -0.9851657748222351, -0.13971543312072754, 0.7332141995429993, -0.037382014095783234, -1.0279980897903442, -1.1702162027359009, -1.1008529663085938, -1.9048752784729004, 0.7576310038566589, 0.22070850431919098, -0.39583316445350647, 0.008000037632882595, -1.2482646703720093, -0.2683457136154175], [-1.1017227172851562, -1.0576204061508179, 1.6082632541656494, 0.030184710398316383, -0.8879737257957458, -1.614120602607727, -0.19718308746814728, -0.27800068259239197, -0.04072152450680733, -1.6714969873428345, -0.3182338774204254, -2.1477389335632324, -0.7330158948898315, -0.005355405155569315, 1.8959238529205322, 0.2756197452545166, -0.9362096786499023, 0.7293429374694824, 0.6769238114356995, -0.6080446839332581, -0.5187993049621582, -0.7247598767280579, -0.6273903846740723, 0.1540777087211609, 0.7051290273666382, -0.2733825147151947, 0.18057960271835327, -0.5713760852813721, -0.9785845875740051, 1.2267160415649414, -0.41542908549308777, -0.2620895504951477, 0.6912726759910583, -1.9221409559249878, -1.5205318927764893, -1.559099793434143, 0.3891369700431824, 0.25041818618774414, -1.3550204038619995, 1.307150959968567, -0.7420241832733154, -0.397434264421463, 0.6303936839103699, -2.0633082389831543, 0.684333860874176, 0.8907655477523804, 0.5049984455108643, 2.232517719268799, 0.45223507285118103, -0.24246521294116974, 0.14197075366973877, 0.3819805383682251, 1.0933432579040527, 0.09318815171718597, 0.4797276258468628, -0.05694323405623436, 1.394769549369812, -1.404887318611145, 1.1221600770950317, -1.2098206281661987, 1.2335078716278076, -0.1291518360376358, 0.3139769732952118, -1.882604718208313, -0.5346229672431946, 0.5211700797080994, 0.7083654403686523, -0.15083956718444824, -0.5003969073295593, 1.0795270204544067, 1.2895022630691528, -1.991162896156311, 0.3186534345149994, 2.185910224914551, 0.2049655318260193, 0.3020450472831726, -0.20590811967849731, -1.1571928262710571, 1.0640959739685059, 0.15691691637039185, 0.04409542307257652, 0.2699839174747467, 1.5592373609542847, -0.7030766606330872, -0.17223858833312988, 0.6834828853607178, 0.15772606432437897, 0.23119601607322693, 2.602806329727173, 1.6640095710754395, -0.05012529715895653, 0.011849693953990936, -0.8062455654144287, -2.907801389694214, 0.1457170695066452, 0.32905831933021545, 1.3759692907333374, 0.6357991099357605, 0.5976496338844299, -2.1771178245544434, -0.43569090962409973, -0.2445019632577896, -0.3947463929653168, -0.007310562767088413, -1.2704589366912842, 0.9173642992973328, -2.2258808612823486, 0.6040841341018677, -0.7600571513175964, 1.6378915309906006, 1.599509358406067, 0.39754948019981384, -2.7797887325286865, 0.33010098338127136, 0.650070071220398, 2.2986888885498047, -0.29048195481300354, -0.10370321571826935, 0.13684354722499847, -0.3051246404647827, 1.5170589685440063, -0.4878620505332947, -2.0871691703796387, 0.15899674594402313, 0.2616437077522278, 0.41697803139686584, -1.193395733833313, 0.27926766872406006, -0.899287223815918, -0.7594329714775085, 0.8622235655784607, -0.16288906335830688, -0.520626962184906, -0.4997173249721527, 0.34433481097221375, -2.0441408157348633, 1.0603708028793335, -1.2855198383331299, 0.47754913568496704, -0.17308788001537323, 1.249106526374817, 2.0883877277374268, 0.032261092215776443, -1.40890371799469, -1.005497932434082, 0.6625235080718994, 0.7398548722267151, 0.6099288463592529, -1.281453013420105, 1.4170901775360107, 0.2747202217578888, 0.04911499843001366, 2.7102675437927246, -0.4363853931427002, 0.29757004976272583, 0.9273464679718018, -0.17539991438388824, -0.5112143754959106, -0.6127800345420837, -0.8346420526504517, -1.4356191158294678, -0.31247439980506897, 1.511217474937439, -0.4949280619621277, 0.8732700943946838, 0.6562351584434509, -0.31684160232543945, 1.8867897987365723, 1.4396816492080688, -1.0347869396209717, 0.2439040243625641, 0.8769964575767517, -0.12746316194534302, -0.3399662971496582, -0.4161038100719452, -0.6910907626152039, 0.6865730285644531, 0.10938115417957306, 1.4716455936431885, 1.1955759525299072, -0.5971763134002686, 0.7626440525054932, -0.2512979507446289, -0.38273903727531433, -0.8586605787277222, 0.8584463596343994, -0.1560509204864502, 0.49096962809562683, -0.5121259093284607, -0.864653468132019, -1.5204490423202515, 0.33731335401535034, 0.2696177363395691, -0.19176173210144043, -0.06703072786331177, 0.05335232987999916, -0.6024429202079773, -0.3181014060974121, -2.048617124557495, 0.2800428569316864, 0.6339890956878662, -0.4713427424430847, 1.217050552368164, -1.4227403402328491, 0.6801736354827881, -0.03979349136352539, 0.40397322177886963, -3.1809194087982178, 0.14337404072284698, -0.4581179916858673, -0.8868007659912109, 1.3416019678115845, -1.2822327613830566, 1.9411108493804932, -0.04851781204342842, 1.3522580862045288, 0.4335691034793854, 1.7987290620803833, 0.42947566509246826, -1.6237866878509521, -0.881975531578064, -0.5223701000213623, 0.2987961769104004, -0.17745532095432281, 0.02901117317378521, 1.1550508737564087, 0.1591627597808838, 0.00417869258671999, 0.6891400814056396, -0.1557195633649826, -1.102591633796692, 1.0842370986938477, -0.2522566020488739, 0.4677237272262573, 0.1587865948677063, 2.189276695251465, -0.7745206952095032, -0.7173908948898315, -0.036458805203437805, 1.2948901653289795, -0.8773564100265503, 0.19289351999759674, -0.6239242553710938, -0.6219996809959412, -0.643979012966156, 0.3596177399158478, 0.3335569202899933, 0.8922517895698547, -1.6190325021743774, 2.0898799896240234, -0.5375108122825623, -1.2379189729690552, 1.4733456373214722, 0.6502161026000977, 0.6643973588943481, -1.0069340467453003, 0.05312449112534523, -1.573195219039917, 0.6066203117370605, 0.8012531995773315, -1.0121471881866455, 0.32397395372390747, -0.9472217559814453, 0.31029653549194336, -1.132106900215149, 1.082351565361023, 0.7538252472877502, 1.0097442865371704, 0.1215023547410965, -0.6114729642868042, -1.1727185249328613, -1.4433547258377075, 0.10356202721595764, 1.2699860334396362, -1.0545474290847778, 1.7334914207458496, -1.2903910875320435, -0.40715962648391724, 0.93873131275177, -1.5214436054229736, 1.0974228382110596, -1.048486351966858, 1.3669793605804443, 0.525506317615509, -0.5240169167518616, -1.0365570783615112, -1.4672889709472656, -0.8177262544631958, 0.12179995328187943, 1.065444827079773, -1.6058543920516968, -2.1674532890319824, -0.9830413460731506, 1.2234303951263428, -0.23750706017017365, 1.4310261011123657, 0.2848598062992096, -0.6308448314666748, 0.6192412376403809, -1.2490047216415405, -0.5923003554344177, -0.7093176245689392, -0.2002883404493332, 0.17982447147369385, -0.8475392460823059, -0.6007699370384216, 0.876636266708374, 0.22193263471126556, -0.308228075504303, 0.241864413022995, -0.6657142639160156, 0.13363496959209442, -1.0586693286895752, -0.07939717173576355, 0.6835517883300781, 0.7186127305030823, 0.0015123648336157203, 0.05464703217148781, 0.5616334676742554, -0.26244449615478516, 1.375537395477295, 1.9453911781311035, 0.6881132125854492, -0.7234068512916565, -0.275576114654541, -0.01325733121484518, 0.5457562804222107, 0.6165330410003662, 0.26450955867767334, 0.1224471852183342, 0.4462246000766754, -1.759321689605713, 0.8206567168235779, 0.6645709872245789, 0.6502442955970764, -0.2762284278869629, -0.1545570194721222, -0.5730736255645752, -0.36361023783683777, -2.439171075820923, -0.265553742647171, 0.9247097969055176, -1.2642120122909546, 1.270061731338501, -1.0785273313522339, 2.0079636573791504, 0.9890996217727661, 1.386235237121582, -0.7549316883087158, -0.08398982137441635, 1.258681058883667, 1.1467084884643555, 0.0043393089435994625, 0.16647043824195862, -0.09933869540691376, -0.32050877809524536, -0.42687150835990906, -0.3034965991973877, -0.21192245185375214, -0.2286537140607834, -1.3535062074661255, -0.7446666955947876, 0.21028216183185577, 0.9295254945755005, 0.7170935273170471, 0.7272135615348816, -1.4840811491012573, -1.9810091257095337, 0.3944490849971771, 0.7392833232879639, 0.36648961901664734, 0.1043759137392044, 1.4451428651809692, -0.8958812952041626, -0.9397038221359253, -0.03025211952626705, 0.7245745062828064, -0.03150258958339691, -0.7642173767089844, -0.31351691484451294, -0.07040324807167053, 2.352424144744873, 0.5412011742591858, 0.501310408115387, -0.7048326134681702, 0.40096691250801086, 0.19866757094860077, 1.113012671470642, -0.02719167247414589, -0.07889307290315628, 0.7155113220214844, 0.6967979073524475, 0.523668646812439, 1.3234885931015015, -1.6806790828704834, -0.00984741561114788, -1.3917362689971924, -0.26010745763778687, 1.3976799249649048, 0.48644477128982544, 0.5324872732162476, -1.7086185216903687, 0.5820470452308655, -0.8882138133049011, 0.9534251689910889, -0.05449540168046951, -0.4274325668811798, -0.8986554145812988, 0.6524670124053955, -0.9116456508636475, -0.9149441123008728, 0.12459467351436615, 1.2366620302200317, 0.17522427439689636, -0.8413962721824646, 0.528131902217865, -0.4928496181964874, 0.4136832058429718, 1.700858235359192, -0.7749435901641846, 0.9837002754211426, -1.505772590637207, -0.5854301452636719, 0.45635437965393066, 0.44235461950302124, 0.7305217981338501, -0.19675005972385406, 0.08715510368347168, 0.030480584129691124, -1.4627512693405151, -1.1704648733139038, -0.627384603023529, -0.39934131503105164, 0.1621190905570984, 0.8614519834518433, -0.5094221830368042, 0.095497727394104, -0.49793392419815063, 3.0777902603149414, 0.5482553243637085, 1.0244265794754028, 1.582458257675171, -0.9083878397941589, -0.4277042746543884, 0.5638576745986938, 2.275111436843872, -0.5644856095314026, 0.10998233407735825, -0.15073737502098083, 0.8938866853713989, -0.7224626541137695, 0.7244662642478943, -0.8406982421875, -1.45744788646698, 1.2702147960662842, -0.6985723376274109, -0.8170803785324097, 0.4166855812072754, -0.6420966386795044, -0.8067901134490967, -0.7329519391059875, 1.2108054161071777, -0.23699034750461578, 0.5413134098052979, 0.2187027484178543, 0.015600714832544327, 0.7277793884277344, 0.4539797008037567, -1.3691227436065674, 1.0895565748214722, 1.0982815027236938, 1.2806549072265625, 0.9948225617408752, 1.1404414176940918, 0.30530399084091187, -0.027394752949476242, -1.2505671977996826, -0.8725190758705139, -1.052815556526184, -0.9399973750114441, 0.011379185132682323, -0.45067843794822693, -0.9395026564598083, 0.8406996726989746, -0.17172372341156006, 0.8467947840690613, 1.022804617881775, 1.4148153066635132, 0.33544501662254333, -1.8346893787384033, -0.9410293698310852, 0.003611060557886958, -1.8172626495361328, -0.9062020778656006, 0.8139387369155884, 1.2084333896636963, 0.502792239189148, 0.6864601373672485, 0.6858885884284973, -0.6180532574653625, -0.2964705526828766, 0.37265947461128235, -1.4362144470214844, 0.6794024705886841, -0.37806272506713867, 0.03742663562297821, -1.2179293632507324, 0.5255187749862671, 0.05834139510989189, -1.0441635847091675, 0.17286625504493713, 0.3162221610546112, 1.982128620147705, -0.04908531904220581, 1.3584965467453003, 0.3243561089038849, -0.40974554419517517, -0.08002268522977829, -1.1098310947418213, 1.0086137056350708, -0.43102869391441345, 0.25248992443084717, 0.17688226699829102, 1.7527918815612793, 0.5273228287696838, 1.1311371326446533, -0.24775470793247223, 2.340574264526367, 0.3367878198623657, 0.6284342408180237, 0.019244564697146416, 0.7366964221000671, -0.40217259526252747, 0.08299294859170914, 0.07902579009532928, 1.4445099830627441, -0.18086078763008118, 0.5662665963172913, -1.0631871223449707, -0.10202579200267792, -0.38390710949897766, 0.8347713351249695, 1.2161797285079956, -0.29332005977630615, -1.0719329118728638, 0.4019010663032532, 0.08624131977558136, -0.38371187448501587, -1.027855634689331, -0.4096166789531708, -0.6994612812995911, -1.185712218284607, 2.0962636470794678, 0.4024657905101776, -0.8308306932449341, -0.7460255026817322, 0.49210482835769653, 0.6942839622497559, -2.3200740814208984, -0.5610876679420471, 0.744228720664978, -0.14924611151218414, -1.450756549835205, 1.1234807968139648, -1.217735767364502, 1.8377299308776855, 0.7189964652061462, 0.8701420426368713, 1.2149940729141235, 1.881239414215088, -0.8478911519050598, -0.25027599930763245, 1.4767553806304932, -1.1688852310180664, -1.0525022745132446, -1.3542189598083496, 0.13815610110759735, -0.33517226576805115, 0.4559219479560852, 1.150740623474121, 1.2498449087142944, 1.1155811548233032, -1.3698533773422241, -2.5676727294921875, -0.6683102250099182, 0.3724157512187958, 0.7632649540901184, 1.4435471296310425, -1.5244389772415161, -0.8768532872200012, -0.14067769050598145, 1.8692498207092285, 0.38700568675994873, 0.14107851684093475, 0.3045825660228729, -0.7945631742477417, -0.5405839085578918, 0.9533154368400574, 0.6798900961875916, -0.3784893751144409, 0.6061317324638367, -0.19570642709732056, -1.5422321557998657, 0.5301576256752014, -0.1671612411737442, 1.4145052433013916, -0.38380172848701477, 0.8964481949806213, 2.040323495864868, 0.41418126225471497, 1.049836277961731, 0.6964209675788879, -0.5506098866462708, -0.7492963075637817, 1.048669457435608, -1.5354585647583008, 0.22213944792747498, 0.24961602687835693, -0.7176682949066162, -0.6611905097961426, -1.742051601409912, -1.151508092880249, 0.36210694909095764, -0.11794795840978622, 1.2458727359771729, 0.28913065791130066, 0.26431459188461304, -0.1853797435760498, 1.5650615692138672, -0.09918227791786194, 0.8157951235771179, 2.682703971862793, -0.9348233938217163, -1.0350397825241089, 1.4380145072937012, -1.5219829082489014, -1.7102305889129639, -0.2507028579711914, -1.0397197008132935, -0.14501933753490448, 0.5272341966629028, -0.2871432304382324, 0.4796091318130493, 0.4515802264213562, -1.5331581830978394, -1.2589937448501587, -0.24245958030223846, 0.7895134687423706, -1.3176355361938477, 0.5666388273239136, 0.6196644306182861, -1.7268000841140747, 0.4914432168006897, -0.973641037940979, 0.9187861680984497, -0.1901704967021942, -0.09314987808465958, -1.5303106307983398, 0.8543956279754639, 0.559787929058075, 0.7127156853675842, 0.915050208568573, 1.0512945652008057, -1.1505768299102783, 1.3582487106323242, 1.3240618705749512, 0.35898005962371826, -1.6346629858016968, 0.10644443333148956, 0.3712458610534668, 1.3466061353683472, -0.24720001220703125, -0.7592827677726746, -1.0252947807312012, -0.7860164642333984, 0.8305644989013672, -0.6815925240516663, 1.636956810951233, -1.1193759441375732, -1.1554694175720215, 0.4030543267726898, -0.19593633711338043, -1.0914266109466553, 1.3733209371566772, -0.6118882894515991, -0.19563032686710358, -1.3725264072418213, -1.7233648300170898, 0.8179165720939636, 0.22778770327568054, 0.37027281522750854, 0.5765552520751953, -1.2233816385269165, 2.0649895668029785, -0.9816295504570007, -0.8785192966461182, -0.19817472994327545, -1.0199064016342163, 0.7715716361999512, 1.7642911672592163, -0.2871132493019104, 0.14916856586933136, 0.24920707941055298, 0.4189983904361725, -0.05537799373269081, -0.7093361020088196, 0.3431798219680786, -0.8280933499336243, -0.08987102657556534, 1.5474870204925537, 0.28684696555137634, -0.4910162091255188, 0.6085494160652161, 0.000763775547966361, -1.1512830257415771, -0.08579212427139282, -0.8011137247085571, -1.496437430381775, 0.4140605926513672, 0.9219726324081421, 1.5148512125015259, 0.5559617877006531, -2.4659030437469482, -0.7964401841163635, -0.5621187686920166, 0.021966662257909775, 0.28983375430107117, -1.0257632732391357, 0.14894334971904755, 1.532870888710022, 1.4928877353668213, -0.8696364760398865, 0.5140180587768555, 1.398391842842102, 1.433843731880188, 0.19520726799964905, -0.47082090377807617, -2.3270726203918457, -0.5227574706077576, -0.44200775027275085, -0.0632370114326477, 0.6776229739189148, -2.006880283355713, -0.6205435991287231, 0.8514008522033691, -0.838470995426178, 1.6543067693710327, -0.05251822993159294, 1.013481616973877, -1.432146668434143, 1.2076845169067383, 0.7796961665153503, -1.1347317695617676, -1.747874140739441, -0.19732868671417236, -1.1970597505569458, -1.633724331855774, -0.13664385676383972, 0.49500223994255066, -0.3050767481327057, -0.2876383066177368, -0.20631995797157288, 0.8268775343894958, 0.09075289964675903, 1.2288334369659424, -0.9923290014266968, -1.8365001678466797, 0.6145394444465637, 1.6842215061187744, -0.7454203367233276, 0.07518932968378067, -0.7455126643180847, 0.8636410236358643, 0.4455195665359497, 0.2766759991645813, -0.9704294204711914, -0.8755236268043518, -0.8090836405754089, -0.4784250557422638, 0.10643552243709564, -0.5084522366523743, 1.3338416814804077, 1.248278260231018, -0.3544768989086151, 0.5848868489265442, 0.7330153584480286, 0.2655595541000366, -1.0043020248413086, 2.2490577697753906, -0.5705165863037109, 0.38188159465789795, -0.3072499632835388, -0.06024907901883125, 0.738832950592041, 0.37300488352775574, 0.18126027286052704, 1.620327353477478, -0.6869621872901917, 1.4865756034851074, -0.5294981002807617, -0.13361431658267975, -0.6456397771835327, 2.4587202072143555, 0.8749914169311523, -0.10559682548046112, 0.5653900504112244, 0.9414423704147339, -0.49691498279571533, -0.5346300601959229, 0.2790824770927429, -0.9531206488609314, -1.147355556488037, 0.24819710850715637, -0.5800572633743286, -1.002716064453125, -0.7238070368766785, 0.5621721744537354, -0.35352057218551636, -0.3964765965938568, -0.6282421350479126, -0.6389961838722229, -0.5617349147796631, 0.2224636822938919, -0.08102262765169144, 2.1557767391204834, 1.6883865594863892, -0.2956794500350952, 1.1934374570846558, -0.10228049010038376, -0.08033687621355057, -0.747291624546051, 1.0438834428787231, 1.2725886106491089, 0.9078463912010193, -0.5155677795410156, -3.1400489807128906, 0.30258867144584656, -0.09809349477291107, -0.43492361903190613, 0.27757394313812256, 0.6449954509735107, -0.48598575592041016, -1.2010935544967651, -0.41091904044151306, 0.15650315582752228, 0.360441654920578, 1.0434932708740234, 0.1624293327331543, 0.1321750432252884, -0.6115243434906006, -1.393660306930542, 1.3864052295684814, -1.0308160781860352, 0.09635070711374283, -1.385197401046753, -0.571372389793396, -0.14541484415531158, -0.6891847848892212, 0.2910642623901367, 0.23043127357959747, -0.7259687781333923, -0.3713371455669403, 0.3679963946342468, 1.401824951171875, -2.010990619659424, -0.16967418789863586, -2.8814685344696045, 1.1158291101455688, 0.09387803077697754, -1.0719012022018433, 0.5626629590988159, -0.8878751993179321, 0.36785778403282166, 0.0901944637298584, 0.2574527859687805, -0.7785711288452148, -0.3010040819644928, -0.24794939160346985, -0.4517291188240051, 2.180049419403076, -0.344631552696228, 2.188248634338379, -0.7942981123924255, -0.7557862997055054, -0.24679312109947205, 1.8793574571609497, 0.6303772330284119, -0.35174357891082764, -0.2387837916612625, -1.5357967615127563, -1.3399149179458618, -0.6067007780075073, 2.8392019271850586, -0.41394752264022827, -1.6365458965301514, 1.7563837766647339, -0.6448413729667664, 0.22473150491714478, -0.5688940286636353, 0.22100518643856049, -0.19756273925304413, -0.7054239511489868, 0.3915906250476837, -1.236963152885437, -0.3939838111400604, 0.6955697536468506, 0.9825247526168823, 0.6558880805969238, -0.5832358002662659, -0.17345523834228516, 0.5801299810409546, -0.7797462940216064, -0.33304259181022644, 0.1946113109588623, 0.49145954847335815, 0.4844416081905365, 0.9047743678092957, 1.332424283027649, 0.5702567100524902, -1.1860240697860718, -1.1138173341751099, -0.19077451527118683, 0.8195540308952332, 0.04408401995897293, -0.2242238074541092, 0.5329052209854126, 0.4378131628036499, 1.4424742460250854, 0.28187817335128784, 0.5833297967910767, 1.2855006456375122, -0.6706903576850891, -0.39182522892951965, -0.7262428998947144, 0.6171827912330627, 1.0547205209732056, 0.6296107172966003, -0.11404645442962646, -0.09668401628732681, 1.9225550889968872, 0.8997446298599243, 1.3661731481552124, -0.16567383706569672, -0.2613406479358673, -2.6793649196624756, -0.38745206594467163, 0.664863646030426, -0.00725348899140954, 0.3715244233608246, -0.9547092914581299, 0.24402010440826416, 0.1946752369403839, 0.6290462017059326, 0.871696949005127, -0.6109492182731628, 1.0198899507522583, 0.1650090515613556, -0.7544149160385132, 0.7622848153114319, -0.3567409813404083, 1.1727876663208008, -0.6683089137077332, -0.6273369789123535, -0.8816826343536377, -0.6759748458862305, 1.3818464279174805, 1.8006256818771362, -0.44172340631484985, -1.9277654886245728, 0.3135126233100891, 1.226150393486023, 0.08638371527194977, -0.01753694750368595, -0.4004141688346863, -0.3132268786430359, -1.0322660207748413, -1.2743802070617676, 1.3499153852462769, -1.470166563987732, -0.24364975094795227, 0.18698953092098236, -0.9973924160003662, 1.137181282043457, -0.8478662967681885, -0.8369268774986267, -0.6546989679336548, -0.7016959190368652, -2.250688314437866, -0.5428151488304138, -0.3714507520198822, -1.1622554063796997, -0.7010524868965149, 1.1289628744125366, -1.0982719659805298, -0.42942678928375244, -1.8123664855957031, -0.09606080502271652, 1.503206729888916, 0.8863494992256165, 2.089137554168701], [0.2755047082901001, -1.6473994255065918, 0.8901063203811646, -0.9682067036628723, 0.15941937267780304, -0.16551421582698822, -0.888965904712677, -0.5353406071662903, 0.31873366236686707, 0.4259684681892395, -0.730637788772583, 0.9438168406486511, -1.3991501331329346, 0.7466993927955627, 1.0353447198867798, 0.12939497828483582, -0.7294957637786865, 1.548014521598816, 2.266801118850708, -0.5942353010177612, 0.07927544414997101, -1.387566089630127, 1.331129789352417, 0.08424431830644608, 0.0014047014992684126, 0.5203930735588074, 0.15282049775123596, 1.252003788948059, 1.2376362085342407, 0.8488122224807739, 1.1705297231674194, 0.12434651702642441, 0.24170006811618805, -2.927635431289673, 0.16298866271972656, -0.17025741934776306, -2.0450658798217773, -0.6636003255844116, -0.08024522662162781, -1.7692139148712158, 1.681317925453186, -2.753516674041748, 0.02305026724934578, -0.4561738669872284, 0.26455414295196533, 0.6563432812690735, 1.2026194334030151, -1.1121909618377686, 1.2185654640197754, -0.9731324315071106, -0.7625057697296143, -1.2829183340072632, -0.891579806804657, 0.8578532934188843, -1.7872501611709595, -2.6717069149017334, 1.392987847328186, -2.033513069152832, 0.6899861693382263, 0.5786416530609131, 0.26524338126182556, -1.494445562362671, -1.4461865425109863, 1.3857771158218384, -0.13938486576080322, 1.9805151224136353, 0.18912217020988464, 0.8601216077804565, 1.1615623235702515, -0.34483861923217773, 0.14892666041851044, -0.582063615322113, 1.2995792627334595, -0.8683193922042847, 0.7746639251708984, 1.015462875366211, -0.7478780746459961, -0.14403562247753143, -1.0943094491958618, 0.17191150784492493, 1.917553424835205, -1.76296067237854, -2.0008544921875, -0.44438377022743225, 0.9989639520645142, -0.1392422318458557, -0.7112094759941101, 0.9161185026168823, -0.18205705285072327, -0.8866699934005737, 1.0942262411117554, 2.1427438259124756, -0.624040424823761, 0.08847209066152573, -1.0886681079864502, -0.06102101877331734, -0.4720901846885681, -2.4146363735198975, -0.9770876169204712, 0.5842065811157227, -0.0920153483748436, -0.3578844368457794, -1.0755465030670166, -0.5626035928726196, 1.4985177516937256, -0.26372310519218445, -2.55954909324646, -0.6424846053123474, -0.20338039100170135, 0.7967498898506165, 0.4268629252910614, -1.0520747900009155, 1.4742625951766968, -0.08940963447093964, 0.833432137966156, -0.2998509109020233, -0.8497397899627686, -0.3174510598182678, -1.192643642425537, -1.313378095626831, -0.38563433289527893, -0.7103147506713867, 0.30354586243629456, -2.3047049045562744, -0.36855849623680115, -0.25582942366600037, 0.6544837355613708, 0.6086694002151489, -0.37447723746299744, 1.2822924852371216, 1.5042638778686523, -0.5659700036048889, 2.070667028427124, -0.20539747178554535, -0.7489296793937683, -1.4529730081558228, -0.21456214785575867, -1.0521125793457031, -0.8854504823684692, 0.6317350268363953, 1.0414321422576904, 0.2763708233833313, 0.5242704153060913, -0.2672128975391388, 0.6098434925079346, -0.3285541534423828, 0.729733943939209, -0.7654974460601807, 1.6575539112091064, 0.9123100638389587, -0.22413823008537292, -1.1086833477020264, -1.4029350280761719, -0.2104063332080841, -1.3341213464736938, -0.8737379908561707, 0.5239574909210205, 2.1676266193389893, -2.0871474742889404, 0.03340240567922592, -1.7850086688995361, 0.05586731433868408, -0.6544774174690247, -0.11979909986257553, -0.18534080684185028, 0.9859002828598022, 0.19862569868564606, 0.30515116453170776, -0.2512657344341278, -0.3498605787754059, -0.4698482155799866, -1.0969187021255493, -0.25260165333747864, 1.7328979969024658, -0.9453307390213013, -0.04329713433980942, -0.8484280109405518, 0.7834450006484985, 1.7080295085906982, -0.011094610206782818, 0.06001649051904678, 1.180605411529541, -0.7849733829498291, 0.14658936858177185, -2.146385669708252, -0.4892071783542633, -1.5494612455368042, -0.2725929021835327, 0.22014985978603363, -0.43603813648223877, 0.4713417589664459, -0.21133840084075928, 1.688414454460144, -1.0607470273971558, 0.015498386695981026, -0.3558606207370758, -1.2289421558380127, 1.0532865524291992, 0.21159328520298004, 0.40017855167388916, 1.5941673517227173, 1.2729756832122803, -0.7226642966270447, 0.2746872007846832, -0.7854292988777161, 1.0229108333587646, 1.4797145128250122, -0.29089468717575073, 0.8552511930465698, 0.8188326358795166, -0.3478304147720337, -1.3568319082260132, -1.944053053855896, 0.4178113639354706, 0.11162741482257843, 1.7301615476608276, 0.9666721224784851, -0.5465726852416992, -0.9986357092857361, 0.016186727210879326, -0.9939420223236084, 1.1983187198638916, -1.048015832901001, -0.1207043007016182, -1.7122489213943481, -0.31325972080230713, 1.9092791080474854, 1.8767004013061523, 0.8306498527526855, -0.8496869802474976, 0.5774204134941101, -1.382994294166565, -0.5754271745681763, 0.43841713666915894, 0.14414387941360474, -0.23444652557373047, 0.46661534905433655, -1.2219339609146118, -0.3587387800216675, -0.2565205991268158, 0.29511722922325134, -0.5418301820755005, 1.6801159381866455, 0.9927792549133301, -1.998051643371582, 1.2844456434249878, -0.5676254630088806, -1.426365613937378, -1.4153281450271606, -0.22948941588401794, 2.902661085128784, 1.557008147239685, 0.7289668917655945, -0.29490622878074646, -0.5901167988777161, 1.0505831241607666, 0.5110182166099548, -1.4549713134765625, -0.4044249653816223, -0.9208014011383057, -0.6230177879333496, -1.046857237815857, 0.37389883399009705, 0.1441790610551834, 0.3935990035533905, 1.806467056274414, 1.2999658584594727, 0.2624788284301758, -0.39289984107017517, -1.071954607963562, 1.2492550611495972, 0.10461410135030746, -0.05033861845731735, 1.0734862089157104, -2.4926555156707764, 0.19239993393421173, -1.2211551666259766, 0.48941925168037415, 0.5426091551780701, 0.5217589735984802, -0.07687258720397949, -0.9422364830970764, -0.42582011222839355, 0.760346531867981, 1.0528621673583984, 0.9077651500701904, -1.0469491481781006, 0.31567201018333435, 1.964520812034607, 0.37744414806365967, 0.13294237852096558, -0.2741518020629883, 1.0540180206298828, 0.3606622815132141, 2.1571836471557617, 0.192245215177536, 1.2264814376831055, -0.7798622250556946, -0.6254801750183105, -0.4961130619049072, -0.6204988360404968, -0.1180097758769989, -0.12212570011615753, 0.5322840809822083, 1.099928617477417, -0.6400368809700012, 1.0926965475082397, -1.5938732624053955, -0.4931929111480713, 0.5618094801902771, -0.5159755945205688, -1.3003425598144531, -0.29082876443862915, -0.07837250083684921, 0.0047371420077979565, 2.3553037643432617, 0.9351096749305725, -0.08342552930116653, 0.8243523240089417, 0.03635797277092934, -0.34183070063591003, 0.4713471829891205, -0.3766426742076874, 0.11211059242486954, -0.06567748636007309, -0.056268803775310516, 0.28470173478126526, -0.4607033133506775, 0.6122907400131226, -1.2226195335388184, 1.0082371234893799, -0.570110023021698, 0.608658492565155, 0.8594657778739929, 0.9543958306312561, -0.14236122369766235, 0.6401711702346802, 1.016785740852356, -0.3977124094963074, -0.8280568718910217, -0.5623873472213745, 1.6670969724655151, 1.5388517379760742, -0.5069137215614319, 0.06839556246995926, -0.6914640069007874, -0.05335405841469765, 0.04813503473997116, -1.9887906312942505, 2.6024794578552246, -0.5493820309638977, -2.1927907466888428, 1.2937461137771606, 0.8209235072135925, 1.168874740600586, -1.2236535549163818, 0.6777039766311646, -0.5184506773948669, 1.0985007286071777, 1.4383010864257812, -0.32332178950309753, -2.3089425563812256, 0.842354953289032, 1.242018222808838, 0.7726160287857056, -0.33816269040107727, -0.743779182434082, -1.5678941011428833, -3.3315482139587402, -0.09618943184614182, 1.6710200309753418, 1.6189367771148682, 0.1565384715795517, 0.5055773258209229, 0.40012314915657043, 0.02099694311618805, 0.48912349343299866, -0.5654196739196777, 1.3181047439575195, 1.9588394165039062, -0.33320629596710205, 1.6562607288360596, -0.9912417531013489, 0.8382925391197205, -1.3376232385635376, 0.039353348314762115, -0.6701050996780396, 0.20581121742725372, 0.23985181748867035, 0.7953908443450928, 0.5688406825065613, 0.6004456877708435, -0.3568147122859955, -1.4753944873809814, -0.7286702394485474, 1.3200808763504028, -1.8648104667663574, -0.05391952022910118, -0.45940107107162476, -0.32903820276260376, -0.248109832406044, -1.5045280456542969, -0.7555298209190369, 0.35403144359588623, 0.11696624010801315, -0.20418758690357208, -0.06379232555627823, 1.5566589832305908, -1.0926350355148315, 0.9172877073287964, -1.8264353275299072, 0.8104759454727173, 1.1851438283920288, 0.2299215942621231, -0.07281913608312607, -0.5368576645851135, -0.08422478288412094, -0.06008808687329292, -1.382507562637329, -1.6700531244277954, -1.0076653957366943, 0.5609285235404968, 0.334351509809494, -0.6393750309944153, 1.4091449975967407, -0.7527847290039062, 1.6988393068313599, -0.7956355810165405, -1.6881402730941772, 0.3733733594417572, 0.5189242362976074, -0.9580898880958557, 0.6119379997253418, 0.014175019226968288, -0.7693175077438354, 0.5042837262153625, 0.10257191210985184, -0.3706638514995575, 0.1445239931344986, 0.8842889666557312, -1.754173755645752, -1.4048209190368652, 0.5656994581222534, -1.2403103113174438, 0.9645617008209229, -0.15273457765579224, 0.3005918860435486, -0.028000513091683388, 0.5938250422477722, 2.071124315261841, 0.23258036375045776, 1.1837961673736572, 1.0496675968170166, 2.253993034362793, 0.7240606546401978, -0.5532326698303223, -0.3050244450569153, 0.7237110733985901, -0.11152011901140213, 0.3584282696247101, -0.19347886741161346, 1.1844184398651123, 2.0059077739715576, -0.2744623124599457, -1.1404802799224854, 0.5778118968009949, 0.3603977859020233, 0.9126628637313843, 0.19695371389389038, -0.9514424800872803, -1.5470844507217407, 0.8774096965789795, -0.4650116264820099, 0.2647019326686859, 0.46251776814460754, -0.7647223472595215, 0.5132229328155518, -0.8767173886299133, -0.5485279560089111, 0.14733946323394775, -0.7279784083366394, 0.11179450899362564, 1.6685009002685547, -0.6547622084617615, 0.7814540863037109, 0.36179065704345703, -0.158060222864151, -0.3184226155281067, -0.09777089208364487, -1.6186240911483765, -1.5081560611724854, -1.6737619638442993, -0.36660057306289673, 2.591977119445801, 0.17073801159858704, 0.9619168639183044, -0.33341866731643677, 0.0765717551112175, 0.7203809022903442, 0.8766298294067383, -1.1749095916748047, 0.4696076810359955, -0.5645918846130371, -0.03856613487005234, 1.1588367223739624, -0.3107649087905884, 2.388601303100586, 1.9127386808395386, -0.09493936598300934, 0.27373364567756653, 0.016171162948012352, -0.00675257109105587, -0.026568302884697914, -0.14285865426063538, 0.5688889622688293, 0.4096061885356903, 1.8034569025039673, 2.3070430755615234, 0.46352753043174744, -0.3663017153739929, 0.6435680985450745, -0.1708536297082901, 0.6957044005393982, 1.0203673839569092, -1.1668602228164673, -0.7615492939949036, -0.23490603268146515, 0.13905537128448486, 0.3227805495262146, 0.0055250441655516624, 0.04826625809073448, -0.11687895655632019, 1.7033125162124634, 0.29601526260375977, 0.5303192138671875, 1.2477535009384155, 0.4910842776298523, 0.9362325668334961, -0.5348426699638367, -0.0676092728972435, -0.45406100153923035, -0.5909358263015747, -0.6394771337509155, 0.7611702084541321, -0.06798209249973297, -1.3701200485229492, 1.8948767185211182, -0.6027980446815491, -1.1605072021484375, -0.3284156322479248, -0.7312681674957275, 0.45110976696014404, -1.6306761503219604, -0.533589780330658, -0.5180773735046387, -0.5345036387443542, -0.6975138783454895, 1.5823125839233398, -1.200913667678833, 0.05154796317219734, 0.1006738692522049, 0.7358992099761963, 1.8883683681488037, -0.07234373688697815, 0.30559903383255005, -0.8896853923797607, 0.8721315264701843, 0.8880847692489624, 0.843131422996521, 0.7817047834396362, 2.202885866165161, 0.8011763095855713, -0.9105526208877563, -0.0015323966508731246, -0.49727359414100647, -1.5906825065612793, -0.007520989049226046, -1.9610880613327026, 1.1217378377914429, 0.7231777906417847, -0.04968802258372307, -0.02676602080464363, -0.06344694644212723, 0.725767195224762, 0.34245267510414124, -0.4674436151981354, 0.5025692582130432, -1.6805003881454468, -0.06240502744913101, -0.6309313774108887, 0.4360334873199463, 1.7525668144226074, 1.0488752126693726, 0.38308703899383545, -0.149366095662117, -1.34099543094635, 0.6046572327613831, -0.5907794833183289, 0.7642274498939514, -1.2316585779190063, 3.29618501663208, 1.4146414995193481, 2.255502223968506, -1.0577646493911743, 2.883615493774414, 0.49786728620529175, 1.5572693347930908, -0.21760401129722595, -0.26150020956993103, -0.26557421684265137, 0.7211673259735107, 1.6923633813858032, 0.05209611728787422, -0.41015660762786865, -1.4420989751815796, -0.42830222845077515, -0.6355928182601929, -0.02188362367451191, -1.1389141082763672, 0.537308931350708, -0.6757129430770874, -1.5062603950500488, -0.08273196220397949, -0.7699564695358276, -0.45201388001441956, -0.6025015115737915, 2.3465189933776855, -0.9185774326324463, 0.13999080657958984, -0.4979543685913086, -0.37503352761268616, 0.32264599204063416, 0.20244435966014862, 0.8346763849258423, 0.04501209035515785, 0.4826732873916626, 0.14644195139408112, -1.0341095924377441, -0.8629278540611267, 0.8468162417411804, -0.5276572704315186, -1.1402767896652222, 0.16609112918376923, 0.2205471247434616, -2.0033624172210693, 0.8751492500305176, -0.43780118227005005, -1.015026330947876, 0.7865764498710632, -0.7288437485694885, -0.3848855793476105, -0.3639625310897827, -0.02957683987915516, -0.9462480545043945, 1.2971704006195068, 0.5529257655143738, -0.5067384243011475, -0.17379742860794067, 0.6526578068733215, 0.9372572898864746, 0.7036638855934143, -0.32522445917129517, -2.2630927562713623, -0.044635314494371414, -0.30609825253486633, 0.01251008827239275, -0.01978093758225441, -0.48396000266075134, 0.9043824076652527, -0.7201207280158997, -0.09166157990694046, -0.900036096572876, 0.8929070234298706, -0.11309565603733063, 0.1650870144367218, 0.8601788282394409, 1.3774266242980957, -0.05584089085459709, 0.389554500579834, 0.5915835499763489, 1.4397906064987183, -0.70587557554245, 0.6494049429893494, 0.43434447050094604, 0.17469452321529388, 0.39481121301651, 0.18471930921077728, 0.5560488700866699, 0.1230543851852417, -0.24782854318618774, -0.5413154363632202, 0.2649120092391968, 0.45858216285705566, -1.1000994443893433, 1.9186294078826904, 0.3218485713005066, -0.02416801266372204, -1.3516751527786255, 0.1320035606622696, -0.16375593841075897, -0.3531796932220459, -0.8733545541763306, 1.852882981300354, 2.300112724304199, -0.34855133295059204, -1.031009316444397, 0.2861400246620178, -0.6770226955413818, 0.34408649802207947, -2.1714231967926025, -0.4597312808036804, -0.7060591578483582, -0.5179415345191956, 2.0688741207122803, 0.5538675785064697, 0.36553817987442017, -0.4577222168445587, -2.0414772033691406, 0.18715469539165497, -1.012310266494751, 0.009023628197610378, -1.1736071109771729, 0.9260599613189697, 0.8396434187889099, -0.00014411039592232555, 0.11166281998157501, 1.5259913206100464, 0.835357666015625, 0.054080765694379807, 1.610617995262146, -0.34697067737579346, 0.3918167054653168, -1.0725980997085571, -1.166662335395813, -0.9491145014762878, -0.039788369089365005, -0.13935382664203644, -0.7260192632675171, -0.07187217473983765, -2.327615261077881, 1.0314967632293701, 0.8890315890312195, -0.2933366894721985, -2.040433168411255, -0.4749719798564911, -0.9705959558486938, -1.426498532295227, -1.1474931240081787, -0.032477762550115585, 1.0860364437103271, 1.326428771018982, -0.7805308699607849, 0.3468743860721588, 0.3991698622703552, -1.3138997554779053, 0.8413114547729492, -0.08631809800863266, 0.4333639442920685, -2.3848507404327393, -0.3882291316986084, -0.5728225708007812, 0.1858207732439041, 1.5937608480453491, 0.3528772294521332, 0.8061556816101074, -1.3829611539840698, -0.31917962431907654, -0.26421114802360535, 0.8230012059211731, 0.576156497001648, 0.7336781620979309, -0.7570914626121521, -0.013075126335024834, 0.5149791836738586, -0.028957171365618706, -0.4465765953063965, -0.2954238951206207, -0.9354514479637146, -0.7670559287071228, -0.5071770548820496, 0.6066558957099915, -1.5321292877197266, -0.7386364936828613, -1.797812581062317, -0.9603756070137024, -0.9268075227737427, 1.4109845161437988, 0.06534732133150101, -1.4062175750732422, -0.5386029481887817, 1.639456033706665, 0.9346572160720825, -0.4633818566799164, -1.6034729480743408, -0.992508053779602, 0.14824405312538147, 1.5322136878967285, 2.7053351402282715, -0.6227177977561951, -0.9285904169082642, -0.5285572409629822, 0.36561620235443115, 0.40289315581321716, 0.4894970953464508, -1.9892604351043701, -1.049306869506836, -0.16715030372142792, 0.2740451395511627, -0.19424055516719818, 1.5093269348144531, 1.1361316442489624, 0.3211994767189026, -0.21586306393146515, 1.2234444618225098, -1.0885392427444458, -2.1572532653808594, 0.394117534160614, 0.10774175077676773, -1.238065481185913, -1.87158203125, -0.2851194739341736, 1.9755580425262451, 1.0929316282272339, 0.3449471890926361, 0.11889784038066864, 0.7142236232757568, 0.884371280670166, 0.52476567029953, -0.0802779421210289, 0.4182289242744446, 0.7725977301597595, -0.1764180064201355, 0.33802151679992676, -0.18848872184753418, 0.997673749923706, 0.46338313817977905, -0.14986884593963623, -0.7492674589157104, -0.04075109586119652, 0.7518989443778992, -0.6798452138900757, 0.37712758779525757, 2.137364625930786, 0.2679867148399353, -1.3522320985794067, 0.6675718426704407, 0.9711511731147766, 0.8550028204917908, -0.6599391102790833, -0.7732836604118347, 0.7128863334655762, -0.7501569986343384, -2.3973190784454346, 0.6218621730804443, -2.7695682048797607, -1.1233991384506226, 0.45449936389923096, 1.8886529207229614, -0.9651056528091431, 0.5851377248764038, 0.24103638529777527, -0.47797420620918274, 1.9634850025177002, 1.045893669128418, -0.21972234547138214, 2.1767077445983887, 0.0661674216389656, 0.7809411883354187, 1.3850153684616089, -0.4204648733139038, -1.2229191064834595, 0.13001295924186707, 1.1035215854644775, -0.5288558602333069, -3.4480364322662354, -1.1512011289596558, -0.4259912371635437, -0.06564803421497345, -0.5534917116165161, -0.5997348427772522, -1.7952792644500732, -0.5339038968086243, 1.478955626487732, -0.9666846394538879, -0.23839622735977173, -2.8096389770507812, -0.04583068564534187, -1.0499444007873535, -1.1856791973114014, 2.051274061203003, 0.6258468627929688, -0.16414329409599304, 0.6891724467277527, -0.488599956035614, 0.5662954449653625, 1.2671172618865967, -0.7763356566429138, -0.01463056169450283, 0.6403887867927551, -0.8397989273071289, 0.198726624250412, 1.8756589889526367, 1.2862462997436523, 0.7082091569900513, 0.03878145292401314, -0.7217236757278442, -0.29075920581817627, 0.9193675518035889, -2.2506308555603027, -0.2918715476989746, -1.3153338432312012, -0.04584239423274994, -0.5069127678871155, 1.1226880550384521, 0.11212363094091415, -1.7352323532104492, 0.15843331813812256, -1.106143593788147, -0.31447678804397583, 0.684312641620636, -1.7603918313980103, -1.5606361627578735, 0.40217557549476624, -0.44298824667930603, 0.21867018938064575, 2.0271565914154053, -0.662071704864502, 0.005055003333836794, 2.095005750656128, 0.1575414389371872, -0.687650740146637, 0.5313444137573242, -0.1707378327846527, -1.0701228380203247, 1.894842505455017, 0.9443825483322144, -1.462936520576477, 0.8100711107254028, 0.16689880192279816, 0.9235665798187256, -0.4768737852573395, -0.23375123739242554, 1.1422117948532104, 0.4717932641506195, 1.251176357269287, -1.8760817050933838, -1.4580315351486206, 0.08976754546165466, 1.9963362216949463, 1.1292293071746826, -1.9882004261016846, -1.0755640268325806, 1.2426722049713135, 0.18933160603046417, 0.5301005244255066, 0.22327499091625214, -1.5618537664413452, -1.0792007446289062, 0.40050411224365234, 0.4994148313999176, 0.25034448504447937, -0.38756680488586426, -2.204366445541382, 0.07940097153186798, 0.4585583806037903, 1.8223625421524048, 0.7609688639640808, 1.0545645952224731, 0.7783382534980774, 0.5337920188903809, -0.09574510157108307, 0.02554977498948574, -0.25092047452926636, -0.20208919048309326, -1.3021713495254517, 0.5832098126411438, -0.22137503325939178, 0.019548552110791206, -0.281386137008667, -0.15512333810329437, -0.36687171459198, -0.3097897469997406, -0.6051232814788818, -0.4981132447719574, 1.908115029335022, -0.8038796186447144, -0.18700608611106873, 0.048578258603811264, -0.6981682181358337, 0.7464350461959839, 0.1668112874031067, 0.25196942687034607, 0.24167731404304504, -0.1132008284330368, 0.7126010060310364, 0.7393627166748047, -0.31370434165000916, 0.27884915471076965, -0.7466371655464172, -0.8195913434028625], [0.22310329973697662, 2.783625364303589, -1.145862102508545, -1.2313185930252075, -0.4197582006454468, -1.0527547597885132, -0.03679152950644493, -0.6516016125679016, -1.5094860792160034, -1.0565812587738037, -1.0452508926391602, -0.7431007623672485, -0.6863699555397034, -0.3158511221408844, 1.4789938926696777, 0.4594786763191223, -1.8954403400421143, 0.0009651828440837562, -0.6611770391464233, 1.1895560026168823, 0.20690672099590302, 0.6068960428237915, -0.0628330186009407, -0.2663516104221344, -0.8555721640586853, 0.06798160821199417, 0.836237370967865, 1.3770877122879028, -0.5871281623840332, 1.4248462915420532, 3.596900701522827, -0.5201796293258667, -1.0897376537322998, 0.5897313952445984, 0.00418309448286891, -1.529749870300293, 1.2243330478668213, -0.4098697304725647, -2.5211691856384277, 1.5197794437408447, 0.41776686906814575, 0.5382606387138367, -0.5913664698600769, -0.6301822662353516, 0.4362656772136688, -1.0137988328933716, -0.4096909463405609, -0.08421309292316437, -1.0787602663040161, -1.1039307117462158, 0.24691079556941986, -1.8010998964309692, -0.543068528175354, -0.06110699474811554, 0.17037765681743622, 0.6108333468437195, -0.6635602116584778, 1.2915656566619873, -0.07495428621768951, -1.2149511575698853, -0.028583355247974396, -0.7596908807754517, 0.7551576495170593, 0.40298959612846375, 0.34869712591171265, -1.7078020572662354, 0.8119400143623352, -0.399886816740036, -1.6374093294143677, 0.8556140661239624, -0.5664001107215881, 1.509091854095459, 0.6774457097053528, 0.04354644939303398, -0.9621356129646301, -2.4188473224639893, -1.220001220703125, 0.4372413456439972, -0.8425037264823914, -0.30785202980041504, 0.06179424747824669, 1.43934965133667, 0.269165962934494, -0.6166926622390747, 0.3865182101726532, -0.025849202647805214, -0.10675636678934097, 0.27321740984916687, -0.8514410257339478, 0.5412992835044861, -0.03660273551940918, 1.5104445219039917, -0.19241781532764435, -1.0614495277404785, 1.4684443473815918, 0.21881994605064392, 0.47801196575164795, 0.9935798645019531, -1.0891740322113037, -0.589843213558197, 0.9105088114738464, 0.5094031095504761, 0.8357328176498413, 1.062414288520813, -1.0696648359298706, -1.8824598789215088, -0.24690330028533936, 0.27474239468574524, 1.2787141799926758, 0.022662878036499023, -0.15686549246311188, -0.22305741906166077, 0.5303850173950195, 0.3287197947502136, 1.8622740507125854, -0.18593749403953552, 0.45302248001098633, -0.8458173274993896, 1.2804484367370605, -0.4524202048778534, 0.09538537263870239, 1.1638363599777222, 0.0027893378864973783, 0.524031400680542, -0.06553485244512558, -0.8466998338699341, 0.9194185733795166, -0.0766817256808281, 0.11919533461332321, 0.3435596823692322, 1.3865549564361572, -0.742132306098938, -1.7045375108718872, -1.577239751815796, -0.1439690738916397, 0.6484994292259216, 0.6143821477890015, -0.6852074861526489, 0.09833797812461853, -1.1814205646514893, -1.3511126041412354, 0.4998927116394043, 1.2650518417358398, 0.4914175868034363, 1.1019479036331177, 1.309497594833374, -0.9828495383262634, 1.469247579574585, 0.5233607292175293, -0.4094576835632324, -1.058071494102478, 0.8836091756820679, 0.3170467019081116, 0.8202589154243469, -0.7419431805610657, 1.0914779901504517, -1.7703298330307007, 1.1197715997695923, 0.6584722399711609, 1.70025634765625, 1.1167482137680054, 0.1443118453025818, 0.5983355045318604, 0.08934718370437622, -1.3382115364074707, -0.9860383868217468, 0.6727520227432251, -3.5210535526275635, 1.023319125175476, 1.1397852897644043, 0.9398397207260132, -0.3835313320159912, -3.1766300201416016, 0.5906016826629639, 1.9542508125305176, 0.24742305278778076, -1.6445542573928833, 0.5161364078521729, -1.7451730966567993, -2.0748162269592285, -1.4221211671829224, -0.06794574111700058, -0.05670517310500145, -1.2582659721374512, 0.6259925365447998, -0.938685417175293, -1.3628863096237183, 0.45563042163848877, -0.02114461548626423, -0.158604696393013, 0.5104620456695557, 0.26180070638656616, 0.5144376158714294, -0.15860393643379211, -1.740843415260315, 0.018214059993624687, 0.09902065992355347, 1.0540450811386108, -1.2030200958251953, -0.31314265727996826, -1.8156198263168335, -0.40975719690322876, -1.4288324117660522, 1.2266708612442017, 1.2651652097702026, -1.327437162399292, -1.12733793258667, 0.10009375214576721, -0.8156205415725708, -0.10621935874223709, 0.40889984369277954, -0.2186233550310135, -0.06769881397485733, -0.895762026309967, -1.698685884475708, 0.33168119192123413, 2.455585479736328, -0.37203502655029297, 0.10037379711866379, 0.37386617064476013, 0.4000137448310852, 1.3130854368209839, 0.9255906939506531, 1.3975298404693604, 0.45607107877731323, 0.7447671294212341, 1.647573709487915, -0.4098047912120819, -1.175612211227417, -0.5329709649085999, 0.712648868560791, 0.05371483787894249, 0.6074575185775757, 0.7532601952552795, -1.6267898082733154, 0.784322202205658, 0.12204556167125702, 0.02173573710024357, 1.329040765762329, 0.2774870693683624, 1.1582391262054443, -0.8774839043617249, -1.473084568977356, 0.6422220468521118, -0.1064985990524292, 0.2924109995365143, 0.44113606214523315, -0.8570759296417236, -1.0202699899673462, -0.6154668927192688, -0.5162931680679321, 0.9306546449661255, -0.1983085572719574, 0.6395651698112488, -2.0365710258483887, -0.1254630833864212, -0.9528089761734009, -1.4647902250289917, 1.386785864830017, 0.7871219515800476, 0.40828239917755127, -0.45678216218948364, 0.6127385497093201, 0.11441250890493393, 0.20770658552646637, -1.0742392539978027, -0.9470722675323486, -0.23712462186813354, -1.6253604888916016, -0.2976313829421997, -0.5665375590324402, -0.3888508975505829, -0.46188199520111084, -1.0495033264160156, -0.003981694579124451, 0.24239422380924225, -0.0015382649144157767, -0.2822399437427521, 1.794786810874939, 0.9420936703681946, -0.19418363273143768, 0.29203057289123535, -0.4516567587852478, -1.1985127925872803, 1.575606346130371, 1.5772597789764404, -0.25976794958114624, 2.85274338722229, -0.29918789863586426, -0.3619895279407501, -0.8603106737136841, -0.18564167618751526, -0.6025654077529907, -1.329505205154419, -1.2258485555648804, 0.9602015018463135, 0.3428212106227875, -2.258228063583374, 0.15583200752735138, -1.1358667612075806, -0.7466588020324707, -0.3591337203979492, -1.7255197763442993, 0.3315538167953491, -0.9522073268890381, 0.28451141715049744, -1.8300418853759766, -1.072270393371582, -0.6545747518539429, 1.0853772163391113, -1.8140294551849365, 1.4190713167190552, -0.44026434421539307, -1.6531122922897339, -0.22751915454864502, 2.2659389972686768, 0.026182284578680992, 0.6153472065925598, -0.4751865267753601, -1.1132084131240845, 0.32183951139450073, 0.6458008289337158, 1.242021918296814, -0.5160649418830872, 0.4217791259288788, -0.8753541111946106, -0.1285048872232437, 0.1704893857240677, 0.5835031270980835, 0.9617099761962891, -0.2789842188358307, 0.2734456956386566, -0.7099520564079285, 0.6345604658126831, 0.4246525466442108, 0.3007027506828308, 0.904070258140564, -1.8450435400009155, -1.5429682731628418, 0.09890417754650116, -0.14999645948410034, -1.2529946565628052, 1.0423650741577148, -0.4738142788410187, -1.4400533437728882, -0.2213197499513626, 0.1271246075630188, -2.422513723373413, 0.3256123661994934, -1.248221755027771, 0.8314716219902039, -0.8116121292114258, -1.2133382558822632, 0.6761073470115662, -0.9340616464614868, 0.9649096131324768, -0.5179505944252014, -0.053188957273960114, -0.5293473601341248, -1.1658438444137573, -0.535781979560852, 1.5533565282821655, -0.7290471196174622, 0.053407568484544754, -0.18020081520080566, -0.7589163780212402, -0.3950846493244171, 0.4572000801563263, 1.9279522895812988, -0.6715718507766724, -1.3191859722137451, -0.39570558071136475, 0.34440717101097107, -0.08207432180643082, -2.524752140045166, 0.466734379529953, -1.5476152896881104, 1.445062518119812, 0.7257500290870667, 0.4192792773246765, -0.14082378149032593, 1.213732123374939, -0.685211718082428, 0.7401947975158691, 1.5405797958374023, 0.2564881443977356, 0.3146383464336395, 0.6076422929763794, -0.297564297914505, 0.46390190720558167, -0.1099395900964737, 1.2471520900726318, 0.841175377368927, 1.1659209728240967, 0.9676383137702942, -1.0348488092422485, -0.3190585672855377, 0.34671294689178467, -0.653371274471283, -1.4039138555526733, -1.3155906200408936, 1.2048968076705933, 0.7727272510528564, 0.41450363397598267, 0.8330717086791992, 0.9677489995956421, 1.6120498180389404, 1.2318975925445557, -0.24985696375370026, -0.1964786797761917, 1.0219643115997314, 0.04690203815698624, 0.25749504566192627, -0.7665044069290161, 1.770634651184082, 1.133653998374939, -0.07130742818117142, 0.11760556697845459, 0.16424337029457092, -0.7174496054649353, -2.526543378829956, 0.01413465291261673, -0.3071679174900055, 0.6036761403083801, -0.5644954442977905, 0.9760276675224304, 0.5432092547416687, -1.8246608972549438, -0.5008451342582703, -0.445533812046051, 0.5496605634689331, -0.8500977754592896, 1.0389301776885986, 0.33486616611480713, -0.4770801067352295, 0.19775715470314026, 0.17286038398742676, 1.0713602304458618, -0.7867338061332703, -1.2906802892684937, 0.007982004433870316, 0.5909814834594727, -0.19444745779037476, -0.20931066572666168, -1.0562245845794678, 0.5295717716217041, -1.6077924966812134, -0.008684419095516205, -1.1464959383010864, 2.3267836570739746, 1.1823376417160034, 0.9973583221435547, 0.9122127890586853, 0.278580904006958, 0.21892984211444855, 0.8175541162490845, 0.801551342010498, -0.9908525347709656, 0.9704023599624634, 0.681855320930481, 0.09257110953330994, -0.19772745668888092, -0.11255118995904922, -0.15223020315170288, -0.2988106310367584, 0.8406001925468445, 0.8030357956886292, 1.9427720308303833, -1.5945749282836914, -0.9063591361045837, -0.6363282799720764, -0.3835832476615906, 0.23228900134563446, -1.8975324630737305, -0.2043166607618332, 0.3598169982433319, -0.7630214095115662, -0.6960256099700928, -0.832364022731781, -1.679802417755127, -1.1587305068969727, -1.2376633882522583, 0.7603338360786438, -0.5152391195297241, -0.3539426028728485, -0.3233729898929596, 1.0992686748504639, 0.8522993922233582, 1.1322219371795654, -0.33242496848106384, -0.6099848747253418, 0.8340336084365845, 1.2653287649154663, -1.309960961341858, 1.4325953722000122, -0.9772282242774963, -1.9459773302078247, 1.519482135772705, 0.21667338907718658, -0.36663001775741577, 0.2789849042892456, 0.887949526309967, -0.9005399942398071, 1.8830870389938354, -1.3815412521362305, 0.0845196321606636, -0.8391152024269104, -0.8527866005897522, -0.8135464191436768, -0.0882948786020279, -1.803401231765747, 0.058536142110824585, -1.1159502267837524, -0.20287978649139404, -0.007099493872374296, -0.03181219473481178, 0.8788185715675354, -0.7467495203018188, -2.2151641845703125, 0.7019505500793457, 1.4271304607391357, 0.9065920114517212, -0.09242239594459534, -1.1188350915908813, -0.625283420085907, 0.07258575409650803, -1.323208212852478, -1.1714484691619873, 0.8831167817115784, -3.0925137996673584, -0.5866579413414001, -1.3154317140579224, -0.9341928362846375, 0.8089324831962585, -0.970557451248169, 0.9991579055786133, -0.28841522336006165, -1.2172608375549316, 0.5461425185203552, -1.216845154762268, 0.5130941271781921, 0.05415540188550949, -0.21418187022209167, 0.11584615707397461, 0.2985980808734894, -0.29250723123550415, -1.3580418825149536, -0.4769783914089203, 0.9331315755844116, 1.482443928718567, 0.86864173412323, -0.37997761368751526, 0.16409754753112793, 0.597248911857605, 1.090529203414917, 0.14322930574417114, 0.8803246021270752, 1.0475246906280518, 1.7397582530975342, -0.7341839075088501, -0.8231613636016846, 3.001171350479126, -0.20580513775348663, 1.2284845113754272, -0.8406364917755127, 0.9355305433273315, -0.3466491997241974, 0.4738679826259613, -0.2663795053958893, 0.6499378085136414, 0.24608327448368073, -0.12035240232944489, -2.6625559329986572, -1.1733243465423584, -0.41231492161750793, 1.5763440132141113, -1.2031240463256836, 0.3121168911457062, 1.0474613904953003, -0.0495007298886776, 1.3725674152374268, -1.0891177654266357, 1.4924770593643188, -0.7538771033287048, -0.07329802215099335, -0.5014688372612, 0.40552109479904175, 0.4797302186489105, 1.4571431875228882, -0.02582021988928318, 1.6855233907699585, 0.0817667618393898, 0.7802190184593201, -0.3724610507488251, -1.0566486120224, -0.2433886080980301, -0.3317510783672333, -0.7386199831962585, 1.7476099729537964, 1.9392861127853394, -0.11442981660366058, -0.21684865653514862, -1.363481879234314, -0.4808981120586395, 1.1685000658035278, -0.13758651912212372, -0.010377015918493271, -1.6436935663223267, -0.40040603280067444, -0.12469495087862015, 0.9090530872344971, 1.1823910474777222, -1.6159393787384033, -1.4340358972549438, -1.2540487051010132, 1.1642805337905884, 1.7055366039276123, 0.7679646611213684, -0.22376585006713867, -1.7431604862213135, 0.4652685225009918, 0.29613643884658813, 0.03803146630525589, -0.4626152217388153, 0.18496589362621307, -0.1597449630498886, 0.15658003091812134, -1.1134159564971924, -0.5837835669517517, 0.047748442739248276, -0.9036511778831482, -1.758829116821289, 2.3116466999053955, 0.8230151534080505, 0.44055861234664917, 2.6595892906188965, 0.31764253973960876, -0.5865694880485535, -0.36132413148880005, -0.6001477241516113, 0.40822702646255493, 0.4949665367603302, -0.9060210585594177, -0.6998425722122192, 1.9608138799667358, 0.6872665882110596, 0.61566162109375, 0.09157702326774597, -0.8848680853843689, -0.2335907518863678, 0.12118890136480331, -0.7218650579452515, -0.37074488401412964, -0.5593119859695435, -0.7348623871803284, 0.2795155346393585, 0.5848376750946045, -0.4268885850906372, -0.5510365962982178, -1.239565372467041, 1.883444905281067, 1.488742709159851, -0.9085843563079834, 0.5813289880752563, -0.017784982919692993, -0.14802046120166779, 1.6174864768981934, 1.5244896411895752, -0.6579082608222961, -0.5568564534187317, 1.1049532890319824, 0.7743667364120483, -0.9588664174079895, -2.4653401374816895, -0.3145252466201782, 0.8033809661865234, 0.3542896807193756, 0.015710601583123207, 0.24466563761234283, -2.3571107387542725, -1.3753002882003784, 1.9695543050765991, -0.16086629033088684, -0.596005916595459, 0.16544166207313538, -1.361629843711853, 0.1870548278093338, -0.3137272596359253, 0.5977780222892761, -1.1663856506347656, -0.5650774836540222, -0.8226048350334167, -0.01582731120288372, 1.255472183227539, 1.1843372583389282, 1.6675697565078735, -0.0012784602586179972, -0.29163748025894165, 2.7895047664642334, 2.623244047164917, 0.8671064972877502, -1.118916392326355, 0.10653326660394669, 0.9934619069099426, 0.4780402183532715, 2.193915843963623, -0.27006593346595764, 0.27580955624580383, -0.8084968328475952, -0.41513365507125854, 0.7538046836853027, 0.6662267446517944, 0.31786301732063293, 1.4350897073745728, 0.2580501139163971, 0.09546700865030289, -0.971112847328186, -0.3785839080810547, 0.6634323596954346, 1.122056484222412, 0.9829954504966736, -0.12324720621109009, 1.2303261756896973, -0.22128330171108246, -0.14637300372123718, 0.2221335768699646, -0.5380520820617676, -1.2691926956176758, -0.2614321708679199, 1.2283586263656616, -0.21167270839214325, 1.1589014530181885, 0.17860561609268188, 0.6220664381980896, -1.1078959703445435, -2.0403757095336914, 1.3350367546081543, 0.10053233057260513, 1.7580361366271973, -0.41200175881385803, -1.251261591911316, -0.1659349650144577, -0.3283434212207794, 0.6809248328208923, 0.8799710273742676, 0.30805352330207825, 0.17906765639781952, 0.9904857873916626, -0.482035756111145, -0.056879084557294846, -1.026320219039917, -0.7378273010253906, -0.0781911313533783, 0.7488787174224854, -0.8380639553070068, -1.118617057800293, 1.63527250289917, -0.025793086737394333, -0.03786177560687065, 0.28227439522743225, 0.1137184128165245, 1.5777571201324463, 0.5325848460197449, 0.8060027956962585, 0.06508179008960724, 1.7715216875076294, 2.112931489944458, -1.0199220180511475, -1.0079597234725952, 0.48295706510543823, 1.560559630393982, -1.4693964719772339, 0.24018047749996185, -0.8342441916465759, 0.36229854822158813, -0.2121163159608841, 0.2332819402217865, 1.5833911895751953, -2.623847484588623, 0.211268350481987, 1.36407470703125, 1.0649892091751099, -1.8945624828338623, -1.3727749586105347, -1.0385994911193848, -0.17398801445960999, -0.12587480247020721, 0.7867639064788818, 0.48240429162979126, 1.9064462184906006, 1.1939880847930908, -0.7509617209434509, 1.0119284391403198, 0.0534466952085495, 0.020129850134253502, -0.7222968935966492, 0.026355287060141563, -0.7066951394081116, 0.2765992283821106, -0.009048884734511375, -0.148188054561615, 0.18213436007499695, -0.3342691957950592, -0.04085748270153999, -1.698184847831726, -0.031675416976213455, 1.7247215509414673, 0.2856147587299347, -0.8530935645103455, 1.5881588459014893, 0.7010844945907593, 0.5800641775131226, -1.1095525026321411, 0.8168977499008179, -0.570902943611145, 0.6046710014343262, -0.5276389122009277, 1.316340684890747, 0.6358340382575989, 0.7363966703414917, -2.0820860862731934, 0.5872260928153992, 0.11061383038759232, -0.39581942558288574, 1.8610588312149048, -1.6488673686981201, -0.6034868359565735, 0.3273126780986786, 0.6508776545524597, -1.4624937772750854, 0.1503238081932068, -0.0012999485479667783, 0.5357922911643982, 0.9732358455657959, 0.49515676498413086, 0.38254615664482117, 0.15325511991977692, 1.177456021308899, -0.10535326600074768, 0.592646598815918, -1.0997130870819092, 0.1111847534775734, -0.6386755704879761, -0.6557860374450684, -1.1612941026687622, -0.49277231097221375, -0.1915130317211151, 2.0541465282440186, 0.007314521819353104, -0.6536502242088318, 0.9891559481620789, -0.3107900023460388, 0.44533607363700867, -1.032291293144226, 1.0641491413116455, -0.14924484491348267, 0.6594263911247253, 0.18025238811969757, 1.8425371646881104, 1.557058334350586, -0.36260244250297546, 0.513685941696167, 1.1575132608413696, -0.6366599202156067, -0.08347092568874359, -1.17131769657135, -0.6495431065559387, 1.6511867046356201, -0.2560155689716339, 0.7212948203086853, 1.127994179725647, 1.1847625970840454, 0.3999158442020416, 0.024570075795054436, -1.2317191362380981, 2.1376171112060547, 0.37464749813079834, 0.9466317892074585, -0.8427207469940186, 0.4092743992805481, -0.6076688766479492, 0.04207300394773483, -0.15027491748332977, 1.022348403930664, 0.18428920209407806, -0.19939638674259186, -0.9427039623260498, 0.2322326898574829, -1.1432890892028809, -0.15831954777240753, 0.7339890003204346, -2.369535446166992, -1.1100389957427979, -0.6540006995201111, -0.9983839988708496, -0.42570510506629944, 0.1850486397743225, -0.35141628980636597, -0.15615689754486084, -0.7284538745880127, -0.9762449860572815, -1.3473994731903076, 1.554205060005188, 0.5845627188682556, 0.4180333614349365, -0.9581342935562134, 1.6557914018630981, -1.1920746564865112, 1.0543453693389893, -0.9748151898384094, 0.428943395614624, 1.4397345781326294, -0.20850178599357605, -0.701344907283783, -0.775431752204895, 1.5601906776428223, -0.22257201373577118, -1.0023000240325928, -1.4289588928222656, -0.2119254320859909, -0.6695331931114197, 0.16551972925662994, 0.38871705532073975, 0.43003612756729126, -0.17224039137363434, 1.2110768556594849, -0.24988135695457458, -0.03521452471613884, -0.6172431111335754, -1.2241677045822144, 0.6826465129852295, 0.8628396987915039, -0.8860936760902405, 1.0491782426834106, 0.2067282646894455, -0.9726303219795227, -0.9982984662055969, -0.048649709671735764, -0.21975618600845337, -0.5025269389152527, -0.11593928933143616, -1.313112735748291, 0.960330605506897, 1.3910831212997437, 2.311448812484741, 1.7879650592803955, 0.2653888165950775, 0.11724818497896194, 0.19155901670455933, -0.8771443367004395, -0.9617388844490051, -0.196912482380867, 0.9187395572662354, -2.20196270942688, -0.9375183582305908, 1.7770495414733887, -1.3366339206695557, -1.570908784866333, 0.10067127645015717, 0.7686063647270203, -1.6290812492370605, 0.3524564802646637, -1.7275829315185547, 0.29849478602409363, -0.35760271549224854, -1.418928623199463, -0.8770679831504822, 1.8325432538986206, -0.032071538269519806, 1.9056111574172974, -0.021649885922670364, 1.8090966939926147, 1.2370671033859253, 0.8233000636100769, -0.5009454488754272, 0.035914141684770584, -1.1189600229263306, 0.7006970047950745, -0.7781744599342346, 0.4096499979496002, 0.2819185256958008, 0.12511152029037476, 0.8364159464836121, 1.051103115081787, -1.3183702230453491, 0.46500763297080994, -0.09096122533082962, -0.05525023862719536, -0.1865399032831192, -0.3443835377693176, -1.444275140762329, -0.36313629150390625, 1.904317021369934, -1.6303695440292358, 0.4172145128250122, 1.0235289335250854, -0.28524908423423767, 1.2024585008621216, 0.852478563785553, 0.04758856073021889], [0.08381262421607971, -0.10919155180454254, -0.30208826065063477, -1.2989170551300049, -0.458008736371994, -1.1558797359466553, -0.8889425992965698, -1.1670799255371094, 0.5369318723678589, 1.001331090927124, 0.7428827285766602, 0.7179350852966309, -0.4455130696296692, 1.298155665397644, 1.9491634368896484, -0.7283722758293152, -0.12998943030834198, -0.2916465699672699, -0.008699208498001099, -0.6049256920814514, 0.3850446045398712, 0.45671895146369934, -0.9124375581741333, -0.4336845576763153, -0.6196609139442444, 0.4537968635559082, 0.02580486796796322, 0.3043830394744873, -0.13412392139434814, -1.3448331356048584, 0.7058077454566956, 1.0021346807479858, 1.4335551261901855, -0.4211749732494354, -0.6179515719413757, -0.10631787776947021, 1.4044792652130127, -0.4425048232078552, 1.495252251625061, -1.9220811128616333, -2.6326053142547607, 0.7569965124130249, -0.8180028200149536, 0.9427728056907654, 1.6835572719573975, -0.17139984667301178, 0.28842756152153015, -0.29573434591293335, 0.005632162094116211, -0.645984947681427, 0.7360205054283142, 0.8511919975280762, -1.4596030712127686, 1.6737700700759888, 0.9353255033493042, 0.4382443130016327, 1.1021387577056885, 0.6241730451583862, 0.5624985098838806, -1.0528165102005005, -1.7476322650909424, 1.0356494188308716, 0.9858768582344055, 0.8485379815101624, -1.1302140951156616, 0.0061433641240000725, 0.47048455476760864, 0.3819171190261841, 0.9318631291389465, -0.8153101801872253, -0.5732935667037964, -0.9729105234146118, -0.6875514388084412, -0.5253104567527771, 0.8431296348571777, 1.411213994026184, 0.05295271426439285, 1.4315298795700073, -1.7754074335098267, 0.21699658036231995, -0.3527727723121643, 0.26241016387939453, 0.18959391117095947, -0.4465768337249756, -0.8984301090240479, 1.3352655172348022, 0.4339955747127533, -0.1416219174861908, 0.3934447169303894, -1.1238113641738892, 1.7502893209457397, -0.3388780355453491, 1.2733558416366577, -0.22607813775539398, -0.5407088398933411, -1.1093494892120361, 0.24353934824466705, -0.9128345251083374, -1.3045098781585693, 0.7412110567092896, -1.3480150699615479, -2.5866167545318604, -0.3972190022468567, -1.3882672786712646, -0.4311673939228058, 1.0926764011383057, -0.5958688259124756, 0.26979753375053406, 0.9923378229141235, -0.40381985902786255, -1.3698623180389404, 1.1550264358520508, -1.142988920211792, -1.1685106754302979, -0.3777894675731659, -0.08684621006250381, 1.3133795261383057, -0.6965405344963074, 1.14817476272583, -0.06586933881044388, -0.7125872373580933, 0.5814135074615479, 0.37698474526405334, -0.9205210208892822, 1.4758955240249634, 0.4557204842567444, 0.6377853751182556, -0.1929892599582672, -0.35568156838417053, -0.8498914837837219, 0.4877498149871826, 0.4722731113433838, -1.0027832984924316, 1.2500804662704468, 0.1928555816411972, -0.7570899128913879, -0.7999370694160461, -0.8111481070518494, 0.4652669429779053, -0.4412088692188263, 1.6478221416473389, -0.8470655083656311, -2.1580841541290283, 0.4280300438404083, -0.024894196540117264, -0.7115048170089722, 0.10453484207391739, 0.362480103969574, 0.7720608115196228, -1.0256649255752563, 0.22855634987354279, 1.7229787111282349, -0.21602921187877655, 0.8124439120292664, 0.30104899406433105, 0.918608546257019, -0.7483342289924622, 0.5237493515014648, 1.1013171672821045, 0.6551723480224609, -0.22116467356681824, -0.021851923316717148, 0.061495471745729446, -0.25970348715782166, -0.9496698975563049, 0.5151777267456055, -1.6155568361282349, -0.18573413789272308, -0.6560552716255188, -1.7628740072250366, -0.29928284883499146, -1.4357558488845825, 0.5589093565940857, -0.9447181224822998, -1.0399588346481323, 1.223111629486084, 1.420796513557434, 0.16263729333877563, -0.48006540536880493, 1.0464147329330444, 0.4171079993247986, 0.3957265317440033, 1.3808135986328125, 0.8885178565979004, 0.12959325313568115, 1.2213051319122314, -0.4912794232368469, -0.11221940070390701, -0.40303683280944824, -1.2746821641921997, -0.399586945772171, -1.282556176185608, -0.6450579762458801, -1.1880816221237183, 0.3019320070743561, 0.5321702361106873, -0.008554958738386631, 1.3066011667251587, -0.06396307796239853, -2.0409255027770996, -0.046298906207084656, -0.7386267781257629, -0.9655576944351196, 0.825270414352417, 0.23395000398159027, 0.041149478405714035, -0.15341588854789734, 1.4559228420257568, -0.6772409677505493, 0.6377017498016357, -0.07541317492723465, -0.45371562242507935, 0.528839111328125, -1.2637498378753662, -0.75367671251297, -1.1115752458572388, -1.4418630599975586, -0.6475553512573242, -0.357988566160202, -1.0934523344039917, -1.7276551723480225, 1.8155078887939453, -1.454653263092041, -1.960321307182312, -1.063232660293579, 0.4424656927585602, -0.12251268327236176, -1.0730037689208984, 0.11087208986282349, 1.1391675472259521, 0.5101764798164368, 0.2282741516828537, -0.9894073009490967, 0.5764829516410828, -0.2937953770160675, -0.13858996331691742, 1.719140648841858, 0.14543882012367249, -0.08668044954538345, 0.07862944900989532, 0.37062200903892517, 0.9203032851219177, 0.045998312532901764, 0.216262087225914, -0.0325162373483181, -0.8357983827590942, 0.9515631794929504, -0.8729633688926697, 0.47342994809150696, 0.6193470358848572, 2.3966634273529053, 0.05772874131798744, -0.3161768913269043, -0.15507113933563232, -0.7060490250587463, -0.37956497073173523, -1.0897115468978882, 0.6774435043334961, -0.4840530753135681, 0.9082799553871155, 0.2541567087173462, 0.4791463613510132, -0.4158136546611786, 0.08896029740571976, 0.5783225297927856, 1.7565587759017944, 0.7188093066215515, 1.5990790128707886, 1.2243613004684448, -0.5397033095359802, -0.315504252910614, -2.0248818397521973, 0.5327261686325073, 1.5695812702178955, 0.5599454641342163, 2.027981996536255, 2.4200503826141357, 0.4576199948787689, -1.4669550657272339, 0.9177305102348328, 0.14643599092960358, -0.0992891862988472, 1.006456971168518, 0.045986127108335495, -0.22314812242984772, -0.41153961420059204, -0.4754827618598938, -0.6538546681404114, -2.4116287231445312, -0.9758326411247253, 0.44953522086143494, -0.3862513303756714, -0.10424123704433441, -1.3207602500915527, 1.1448616981506348, 1.080698847770691, 1.3664145469665527, -0.5295100212097168, -0.14447593688964844, -0.4093503952026367, 0.7008661031723022, -1.0001819133758545, 0.9424731731414795, -0.5366412997245789, -0.627040684223175, 0.14059802889823914, -1.1105284690856934, -2.1450772285461426, 0.07572857290506363, 0.7311879396438599, 0.25660499930381775, 0.06893596798181534, -0.6992049217224121, -0.4954250156879425, 0.8734050393104553, 1.2329334020614624, -0.42305788397789, 2.542032480239868, 0.5226301550865173, 1.3634881973266602, 1.7919199466705322, -0.858630359172821, -1.7268726825714111, -0.39811503887176514, 1.3449710607528687, -0.16987143456935883, -1.4115865230560303, -0.2904857397079468, 0.5365596413612366, 0.8323138356208801, 1.070890188217163, 0.49486756324768066, -0.2666376233100891, -0.22747021913528442, 0.7356014847755432, 0.9230455160140991, 0.5748450756072998, 1.2929675579071045, -0.8710121512413025, 0.3043670952320099, -0.5627824068069458, -0.845863938331604, 0.1687086820602417, 0.2391744703054428, -1.1434670686721802, -0.3110395073890686, 0.28691747784614563, -0.6192007064819336, 0.09010806679725647, 1.4952415227890015, -0.8478034734725952, 0.20792315900325775, -0.5269126296043396, 1.8735319375991821, 1.9840044975280762, 1.1365635395050049, -1.1287360191345215, 0.6508690118789673, 0.7369294166564941, -0.20805075764656067, -0.3773620128631592, -0.6953741908073425, 0.7099000811576843, 0.5160796642303467, 0.4500831961631775, -1.2464268207550049, 0.9342895746231079, 1.2769981622695923, 0.3479977250099182, -0.729260265827179, -0.45737743377685547, -0.9197190999984741, 0.36136314272880554, -1.6871570348739624, -0.351348340511322, -0.504140317440033, -1.0601458549499512, 1.3084373474121094, 0.30850765109062195, -0.5980523228645325, 0.8460385799407959, 1.9799995422363281, 1.257657766342163, -0.40543505549430847, -1.724048137664795, -0.155497208237648, -0.7325294613838196, 0.49155354499816895, 1.2214821577072144, -0.5715846419334412, 0.004024587105959654, 0.5074486136436462, 0.6629177927970886, -0.12361210584640503, -1.5844578742980957, -1.5554519891738892, 0.7074023485183716, -0.5103226900100708, -0.06629741936922073, -0.5376846790313721, -0.09524144232273102, 0.008571433834731579, -1.5433146953582764, 0.3716377913951874, 1.4783523082733154, -0.12811820209026337, -0.02640264295041561, -0.9671645760536194, 0.8962404131889343, 1.0791646242141724, -1.0227649211883545, 1.1778799295425415, -0.5268983840942383, -0.7378293871879578, 0.27890411019325256, -0.11142577975988388, 0.5225880146026611, 1.2729823589324951, -0.6114833950996399, 1.3063244819641113, -1.4005554914474487, 1.1332199573516846, 1.1924813985824585, -0.36306828260421753, 0.3328600823879242, 0.024978991597890854, -0.5182892084121704, -0.8088840246200562, 0.8774809241294861, 0.33740124106407166, 0.3038957417011261, 0.015776880085468292, 0.6457810401916504, -0.8827953338623047, 0.550147533416748, 0.5268939137458801, 0.7832195162773132, -0.008159022778272629, -0.624618649482727, -1.3165946006774902, 0.5647884607315063, -0.5480825304985046, 0.4651983678340912, 0.8413496017456055, 1.2565498352050781, 0.6338056325912476, -0.5628353357315063, -0.3892138600349426, -0.48721736669540405, -2.8735218048095703, 0.7070132493972778, -0.5699511766433716, -0.1564277559518814, -0.649631142616272, -0.1841077357530594, 0.441967636346817, -0.1613989919424057, 1.1308127641677856, -0.6325719952583313, 0.6854410171508789, 1.0937559604644775, -1.747158408164978, 0.7356088161468506, 0.06904123723506927, 0.5778385996818542, 0.01600186713039875, 1.832535743713379, 1.3476096391677856, -0.5805656313896179, -1.5338455438613892, -0.8670932054519653, -0.1304015815258026, 0.2718355059623718, 1.5000081062316895, -0.21641720831394196, 1.4569958448410034, -1.418075442314148, -1.672644019126892, 0.3970639109611511, -1.7007914781570435, 1.5602105855941772, 0.013159903697669506, 1.0188599824905396, 0.589503824710846, 1.060197353363037, -0.9290924668312073, 1.5125364065170288, 0.8457480669021606, 0.7093986868858337, -0.6815937161445618, -0.425275593996048, -0.3214595913887024, 1.4313077926635742, 0.05636100098490715, 0.1695776879787445, -0.34627363085746765, -0.44111427664756775, 0.7208153605461121, 2.1950018405914307, -0.12686419486999512, 0.33344167470932007, -1.148606538772583, 0.2695603370666504, -0.08012053370475769, 0.43255019187927246, -0.752444326877594, -0.6840274333953857, -0.8748045563697815, -1.1792949438095093, -0.5106611251831055, -0.32329460978507996, 0.08731618523597717, 0.9430702328681946, -0.25995633006095886, 2.0179226398468018, -0.03516887128353119, -1.4755300283432007, -0.06770047545433044, -0.06729572266340256, -0.47727444767951965, 0.6379457712173462, 0.058265600353479385, 0.24467569589614868, 0.04122437909245491, 1.0570148229599, -1.1129361391067505, 0.4988507330417633, 0.27414533495903015, -1.1187444925308228, -0.5311123132705688, -0.3642510175704956, 0.9995425939559937, -0.6959818005561829, -1.176845669746399, -0.6987158060073853, -0.7470192313194275, 0.2872208058834076, 0.9584524631500244, -1.7352484464645386, -0.05111728981137276, 0.9591088891029358, -0.23670710623264313, -0.8930927515029907, -1.4242212772369385, 0.06638769060373306, 0.2595981955528259, -0.10759139060974121, 1.0751031637191772, 1.3334217071533203, 0.10246530175209045, 1.2375177145004272, 1.9208118915557861, 2.6386282444000244, -0.3005099594593048, 1.091369867324829, 0.46095049381256104, -1.0546119213104248, 0.7871863842010498, -0.9354169964790344, -0.2742884159088135, 0.2204234004020691, 1.0277550220489502, -0.5794668197631836, -0.8486121892929077, 0.9501050114631653, -0.4501752257347107, -1.4061660766601562, -0.37422898411750793, -0.4409806728363037, 1.627641201019287, -0.43967410922050476, 0.6137014031410217, -0.7552710175514221, -0.998170793056488, 0.8526467084884644, 1.7331652641296387, -0.5596911907196045, 1.0257329940795898, 0.8797512054443359, 0.6978328824043274, -0.4357641935348511, -1.1445952653884888, -0.6290909051895142, -1.0114572048187256, 0.9723750948905945, 1.144580602645874, -0.8011505007743835, -0.0021152878180146217, 0.717136561870575, -0.5804774761199951, 0.47084933519363403, -0.8383512496948242, -1.0430307388305664, 0.11692710965871811, -1.6537442207336426, -1.9678219556808472, -0.36535051465034485, 0.15008166432380676, -0.5799791216850281, 0.39315178990364075, 0.46371757984161377, -0.5701128244400024, -0.9970179200172424, 1.3485016822814941, 0.0943063348531723, 1.5235621929168701, 0.18322564661502838, 0.7153075933456421, -0.35760799050331116, 0.7265182733535767, -0.09692896157503128, 1.527083396911621, -0.11340250819921494, -1.0343412160873413, 1.84189772605896, 0.3527793884277344, -0.8664505481719971, 1.0247681140899658, 0.6023255586624146, -0.2506655752658844, -0.6327649354934692, 1.1787725687026978, -1.0816118717193604, 1.4135583639144897, 0.30051401257514954, -0.9196587204933167, -0.39123237133026123, 0.6533662676811218, -0.6384078860282898, -1.0106606483459473, 1.3361613750457764, 0.5852006673812866, 1.0952671766281128, -0.48396944999694824, -1.3074990510940552, 0.018292393535375595, -0.07328259199857712, -0.1790837049484253, -0.5655954480171204, -0.6209946274757385, 1.0757070779800415, 0.59935462474823, 0.8249518275260925, -1.3823989629745483, -0.532609224319458, -1.5819226503372192, 0.27494314312934875, -2.0130088329315186, 0.35315385460853577, -0.5573011040687561, -0.5002368092536926, -0.3116433322429657, 0.20941990613937378, 0.8299564123153687, 1.37580406665802, 0.12555177509784698, -2.5863051414489746, 0.8572105169296265, 0.3916146457195282, 0.4750029146671295, -1.4398112297058105, -0.7810906767845154, -1.481886863708496, -0.0652850866317749, -0.2604278326034546, -1.1484153270721436, 1.3178036212921143, -0.4660067856311798, -0.3741135001182556, 1.0420174598693848, -0.0796595886349678, 1.4334192276000977, -0.9497420191764832, -0.5263274908065796, -0.6350820660591125, -1.946360468864441, -0.7962906956672668, -1.7231782674789429, 2.6203503608703613, -0.5538768172264099, -1.3073594570159912, 0.43184518814086914, 0.7195631861686707, 0.21592052280902863, -1.3443878889083862, 0.6024230122566223, 0.04845631495118141, -0.6846179366111755, 0.42079833149909973, 0.1669885367155075, -0.9560539722442627, 1.1871716976165771, -0.30389273166656494, -0.06733184307813644, 0.6242545247077942, 0.992891788482666, -0.26115351915359497, -0.0904335156083107, 0.03576020523905754, -0.6811046600341797, -0.006359468214213848, -0.6722024083137512, 0.1744241714477539, 1.3577865362167358, 0.20989958941936493, -1.1670706272125244, 0.44205236434936523, -0.8342893719673157, -0.2013012170791626, -1.798591136932373, -0.20117495954036713, 0.32483145594596863, 0.3566513657569885, 0.9535322785377502, -2.8031840324401855, 1.1699877977371216, -0.021823184564709663, -0.25768807530403137, 0.35999202728271484, 0.14462023973464966, 1.6010011434555054, 0.20226484537124634, -0.036011096090078354, 0.1607806384563446, -1.3606016635894775, 1.3649622201919556, -0.6814743876457214, 0.3474861979484558, -0.2638753354549408, -0.22376973927021027, 0.9234804511070251, -1.4835485219955444, 0.3196098506450653, 1.9046765565872192, -2.1776254177093506, 1.650009036064148, 0.0771026611328125, 0.3130788207054138, -1.0859630107879639, 0.7774618268013, 0.3795429468154907, 0.3169134557247162, 0.8972015380859375, -0.32047057151794434, 1.2240415811538696, -1.943766474723816, -0.2565447986125946, -2.0228168964385986, 0.1798565834760666, 0.46063467860221863, 0.5667805671691895, -0.5700249075889587, 0.7849624156951904, -1.2831586599349976, -0.9055235385894775, 0.5169591903686523, -0.415584921836853, -0.404834121465683, 0.23326249420642853, -0.44583919644355774, 0.788176953792572, -0.41381943225860596, -0.62928706407547, 1.3356821537017822, 1.5784478187561035, -0.14225704967975616, 0.18345396220684052, -0.7286218404769897, 0.09080910682678223, 0.002906432840973139, -0.22911636531352997, -1.5973154306411743, 0.8759024143218994, 1.4604579210281372, -1.2812937498092651, 0.2126343697309494, 0.41597819328308105, -1.026187777519226, -0.9609522819519043, -1.8590214252471924, -1.959421992301941, -0.013675275258719921, -0.22327718138694763, 0.7085555791854858, -1.1478469371795654, -0.5300328731536865, -1.6643141508102417, -0.9507156610488892, -0.13246534764766693, 1.0516494512557983, 1.142322301864624, 0.25283998250961304, -2.1236655712127686, 0.13316984474658966, -0.588034451007843, -2.053461790084839, -0.763052761554718, -0.1305946409702301, 1.631030559539795, -0.6566838622093201, -0.38281914591789246, -0.6032841205596924, -0.5465403199195862, -0.6064065098762512, -2.3646605014801025, 0.9018205404281616, -1.2278105020523071, 0.3125172555446625, 0.43907806277275085, 0.900298535823822, 0.046824850142002106, -0.9606224298477173, -0.8830968141555786, -0.6446077227592468, -1.5072966814041138, -0.38463619351387024, -0.39445391297340393, -0.5939488410949707, -0.3242601752281189, 1.4417719841003418, -0.4895905554294586, 1.140986680984497, 1.7281807661056519, 1.3450181484222412, 0.5371631979942322, 0.05040658637881279, 0.37673935294151306, -2.0319061279296875, -0.586113452911377, 0.5634433031082153, -1.608940839767456, -0.5881451964378357, 0.9711073040962219, -0.9709527492523193, 1.148563027381897, 0.27509576082229614, 0.06475333869457245, 0.5118991732597351, 2.4110963344573975, 1.3514540195465088, -0.08611050248146057, 0.5470743775367737, 0.09052059054374695, 1.6529408693313599, 0.8885876536369324, 1.3257982730865479, 1.734663963317871, -0.2603680193424225, 1.476195216178894, -0.2351037561893463, 1.0979773998260498, 1.9459481239318848, 2.0525076389312744, -0.30419352650642395, -0.6103515028953552, -1.916049838066101, -0.7385702729225159, -0.8196486234664917, 0.5185989737510681, 0.2604326605796814, -0.7541881203651428, -1.1891745328903198, -0.3831305503845215, 1.2045884132385254, -0.6071259379386902, 0.7178804874420166, -0.43361470103263855, 0.5720531344413757, 0.5528534650802612, 0.5074650049209595, -2.4407732486724854, -0.9920876026153564, -0.7997300028800964, 0.5248664021492004, -0.9748296141624451, -0.4804295599460602, 1.0061566829681396, 0.6449765563011169, 1.3363752365112305, 1.2367353439331055, 0.3407643139362335, 1.4218966960906982, -0.49801868200302124, 0.5657111406326294, 0.8697279095649719, -0.7708739638328552, 1.5972867012023926, -0.12745991349220276, -0.2325407713651657, -0.050358958542346954, 0.33079320192337036, -0.10388358682394028, -0.25998950004577637, 1.7368327379226685, 0.3385554254055023, 0.5340891480445862, -0.6120347380638123, 0.23074662685394287, 0.3580714762210846, 0.913863480091095, 1.0185596942901611, 0.797365665435791, -1.2689428329467773, 1.7604289054870605, -0.3002627491950989, -0.52189040184021, -2.4366893768310547, 0.8042685389518738, 0.8434920310974121, 0.00936073623597622, 2.2820541858673096, -0.3481255769729614, -0.47643807530403137, -0.7511190176010132, 1.5277968645095825, -1.0789669752120972, 1.7648056745529175, 0.2096511274576187, 0.2572876214981079, -0.3030710518360138, 1.071253776550293, 1.406205654144287, 1.1429697275161743, -0.8626518845558167, -0.4493322968482971, -0.7122133374214172, -1.466333270072937, -2.152070999145508, 0.30779504776000977, 0.9549359679222107, 1.2253515720367432, -1.4754371643066406, 0.3963131010532379, 1.7519618272781372, 1.2857109308242798, 0.5880524516105652, -0.30415475368499756, 0.23409505188465118, -0.020161915570497513, -0.6613097786903381, 0.10530494153499603, -0.5173701047897339, -1.7334632873535156, -0.7387117147445679, -1.3941724300384521, 1.9128221273422241, 0.1664954125881195, 0.6365925073623657, 1.8884615898132324, -1.597391128540039, 0.4100254774093628, -0.1144070252776146, -1.3507664203643799, -0.02896660566329956, -0.7467756867408752, -1.081068754196167, 0.40577033162117004, 1.3737748861312866, -0.9338625073432922, -1.3966721296310425, -0.2685506343841553, 0.5216713547706604, -0.3627012372016907, -1.8175079822540283, 1.882330298423767, 0.7239327430725098, 1.832870602607727, 1.0344135761260986, -0.6383132934570312, -1.0712864398956299, 0.21866026520729065, -2.2013044357299805, 0.6706751585006714, -0.3664696216583252, -0.09212532639503479, 0.6820241212844849, -0.08204177021980286, 0.09654994308948517, 0.6669950485229492, -0.5548815727233887, 2.474512815475464, -0.9273127913475037, 0.2573966681957245, 0.6014478206634521, -0.5767304301261902, 1.5504393577575684, -1.6389718055725098, -1.2703758478164673, -1.1080440282821655, -0.9740812182426453, 0.9793739318847656, 0.6795205473899841, -0.7376180291175842, -1.9107670783996582, -0.9324283599853516, -0.2984547019004822, -0.6601139307022095, 0.9956907033920288, 1.3115689754486084, 0.08350669592618942], [-1.3913938999176025, -2.5462851524353027, -1.2085068225860596, 0.2963106036186218, -0.44951796531677246, 2.3881804943084717, -1.3257249593734741, 2.179966688156128, -1.5884686708450317, -1.534716248512268, -0.08748777210712433, 0.2062816619873047, -0.320239394903183, 0.24689078330993652, 1.356645941734314, -0.651745617389679, 0.22784696519374847, -0.6594675779342651, -0.8794697523117065, -0.06256328523159027, 0.9922268390655518, 0.916456937789917, 0.098220594227314, 0.65594482421875, 0.12449414283037186, 0.9326435327529907, 1.1854127645492554, 0.4292755424976349, -0.8636547923088074, -0.09860900044441223, -0.12258467823266983, 0.02231253683567047, -0.8867204189300537, 1.5475354194641113, 0.18481948971748352, 0.44202113151550293, -0.08826512098312378, -0.826327383518219, 0.7268127799034119, -0.4526854157447815, -1.0364559888839722, 2.1125845909118652, 0.5366879105567932, 0.03574467822909355, -1.1412160396575928, -1.4370181560516357, -0.4232478141784668, -0.6463541388511658, -1.9672528505325317, 0.6116278767585754, -0.17668114602565765, 1.2795255184173584, 0.950326144695282, -0.8959994912147522, 1.3424906730651855, 0.24519354104995728, -0.17575065791606903, -2.190464496612549, -1.1684174537658691, 0.6340107917785645, 1.0945149660110474, -1.4504649639129639, -0.18273305892944336, 0.5677430033683777, -0.4149407148361206, -0.9652003049850464, 0.19535113871097565, 0.33682677149772644, 0.26284000277519226, -0.867509126663208, 0.0777764618396759, 0.9925047755241394, -0.3880622982978821, 0.6302788853645325, -0.14440391957759857, -0.31447339057922363, -2.2555336952209473, 0.9248753786087036, -0.2714169919490814, 0.39076781272888184, 0.3401700556278229, 1.039929747581482, -0.12190808355808258, 1.3689827919006348, -1.342037320137024, -0.755091667175293, 1.2284867763519287, -1.019142508506775, -1.0634347200393677, 2.378537893295288, -0.7833887338638306, 0.34344035387039185, 0.7653574347496033, -0.8300513625144958, 0.14880657196044922, -0.6392462253570557, -0.6914997100830078, 0.2533365786075592, -0.16663013398647308, -0.1284293681383133, 0.7763607501983643, 0.20540697872638702, 0.18016742169857025, 1.9584088325500488, 1.2992119789123535, -0.17664237320423126, 0.0746590718626976, -0.18535201251506805, 0.43352073431015015, 1.405255913734436, -1.2629188299179077, 0.11494985222816467, 1.3886891603469849, 0.18881955742835999, 0.593693196773529, 0.9548559784889221, -0.11259480565786362, -0.43441042304039, 2.1900634765625, -0.22659598290920258, 0.31464245915412903, 0.7108219265937805, -0.3139512240886688, -0.44818055629730225, 0.8838929533958435, 0.1834937036037445, 0.8609950542449951, 0.0402224026620388, 0.6000540256500244, 1.697712779045105, 0.2159232199192047, 0.031536176800727844, -0.38651642203330994, 2.0740249156951904, 1.0734058618545532, 1.439981460571289, -0.8353437781333923, -0.792298436164856, 0.6457118391990662, 2.6199707984924316, 1.2393871545791626, -1.068725347518921, -1.852862000465393, 1.0850852727890015, -1.1679728031158447, 0.19493383169174194, 2.025286912918091, 0.5950692892074585, -0.7323413491249084, 0.14632996916770935, -0.6835611462593079, -0.3635563552379608, -0.16959786415100098, 1.5656030178070068, -0.3197600543498993, -1.3647421598434448, -1.8638575077056885, 0.8596903681755066, -0.08350539207458496, 2.3231959342956543, 0.8587983846664429, 1.2209267616271973, -1.5310224294662476, 1.1251553297042847, -0.06124565750360489, -1.656353235244751, 0.06441008299589157, 0.9232900142669678, 0.03782131150364876, 2.382052421569824, 1.7954716682434082, -0.101261205971241, 0.8548738360404968, 0.6022883057594299, 0.0889257937669754, 1.7587765455245972, 2.012725830078125, 0.7377869486808777, 0.48768487572669983, -0.7835916876792908, -0.9534227848052979, 1.0072791576385498, 0.07605265825986862, 0.8843581676483154, 1.4873985052108765, 0.3503158688545227, 0.2023942917585373, 0.4694824516773224, -0.840324878692627, -1.786576271057129, 0.8124105334281921, -0.06230157986283302, 0.3793824017047882, -0.645488977432251, 0.24992477893829346, -0.5264695286750793, 0.0048822080716490746, -1.4404772520065308, -0.5112737417221069, 0.3314847946166992, 1.6596144437789917, -0.15040437877178192, -0.12094097584486008, 0.0295957550406456, 1.7297574281692505, -0.7844553589820862, -1.0636088848114014, 0.7886507511138916, 0.5789238810539246, -1.5721378326416016, -0.8781760931015015, 0.5661681890487671, -0.6447309255599976, 1.5980662107467651, 0.49994850158691406, 0.6097110509872437, 0.4037020802497864, 0.49411678314208984, 1.3421804904937744, 1.503009557723999, 1.0874419212341309, -0.42121657729148865, -0.4389097988605499, 0.7660772204399109, 0.5609153509140015, 0.32847341895103455, -0.14738623797893524, -0.59980309009552, -1.6891628503799438, -0.8486879467964172, 1.2406413555145264, -0.2973780333995819, -0.9922472238540649, -0.3313239514827728, -0.40155112743377686, 0.17355087399482727, 1.1394095420837402, 0.3058655261993408, 0.16727648675441742, 0.820687472820282, 0.3874202370643616, -0.38874292373657227, -0.1445545256137848, -1.237879753112793, -0.46196573972702026, 0.4343459904193878, -0.4127911925315857, 1.3350332975387573, -0.9540578126907349, -1.0217981338500977, -0.816249668598175, 0.4680968225002289, -0.4634270966053009, -0.012978121638298035, -0.5618000030517578, 0.33402448892593384, -2.4981985092163086, -0.593748152256012, 1.1760579347610474, 0.684332549571991, 1.2329943180084229, -1.0813705921173096, -0.1551266312599182, 0.6374002695083618, -1.1378191709518433, -0.07197073847055435, 0.008160247467458248, -0.8817163705825806, -0.16249868273735046, -0.8658921718597412, 0.372353196144104, -0.31080830097198486, -1.1277369260787964, -0.2890905439853668, 0.4648907780647278, -0.044248200953006744, 0.7246026396751404, -0.6493862271308899, -1.2012385129928589, 0.38066577911376953, 0.6970525979995728, -0.09685301035642624, -0.15195327997207642, -0.30494558811187744, 0.6706956028938293, -2.0653412342071533, -0.4680989682674408, 0.41376298666000366, 0.7110182642936707, -0.3149510622024536, -0.44641047716140747, -0.6077467203140259, -1.3517076969146729, -1.8026076555252075, -0.3390583097934723, -0.675956666469574, 1.090018391609192, -0.3483090102672577, 1.6145439147949219, -0.4534083604812622, 0.140059232711792, 0.1287173330783844, -0.4241504669189453, 0.043374672532081604, 0.07687457650899887, -0.4746313691139221, 0.2720383107662201, -0.1729351431131363, 1.0843840837478638, -0.8288886547088623, 0.011721567250788212, 0.8368846774101257, 1.6137058734893799, -0.927692174911499, -1.304938793182373, 1.7425827980041504, -1.0322293043136597, 1.9533823728561401, -0.40871021151542664, -0.7408931255340576, -0.6834239959716797, -2.06587815284729, -0.6558197736740112, 1.5255218744277954, 0.8666343092918396, -1.606555700302124, -0.21157684922218323, -1.0670812129974365, -1.7921229600906372, 0.31166720390319824, -1.0094870328903198, -0.4597317576408386, 0.6836608648300171, 0.2080947458744049, 0.5648324489593506, 1.0897272825241089, 0.6138262152671814, -0.05753907188773155, -0.5429175496101379, -0.5618095993995667, -0.9574111104011536, -0.3775166869163513, 0.06763187795877457, 0.5140947699546814, 0.12012029439210892, 1.8254952430725098, 0.3497750461101532, 1.2920515537261963, 0.31761434674263, 1.692949891090393, 0.9154068827629089, -1.574155569076538, -1.3227477073669434, 0.1331944316625595, -0.9320571422576904, 0.5162835121154785, -0.5730568766593933, -1.7524569034576416, -1.2150533199310303, 1.199798345565796, -1.1064050197601318, -0.4073810279369354, -0.8566022515296936, -0.009219680912792683, -0.1523147076368332, -1.0519267320632935, 2.2979793548583984, 1.5626893043518066, 0.24465975165367126, -1.194200873374939, -0.2690890431404114, 0.46405333280563354, -1.1902680397033691, -2.4795219898223877, -1.1097164154052734, -1.3293890953063965, 0.2831113040447235, -0.8053579330444336, 0.3731222152709961, 0.20765094459056854, 1.2687915563583374, -1.6096192598342896, 0.5662230253219604, -1.0310252904891968, -0.7514082193374634, -0.08028069883584976, 2.321666717529297, 0.694969892501831, -0.46073734760284424, 2.4004697799682617, -0.8136311173439026, 0.4994237422943115, 2.1181812286376953, -1.2803503274917603, 0.03751635551452637, 0.06553975492715836, -0.13910353183746338, 1.4109748601913452, -0.8764075636863708, 0.0013066618703305721, 0.12996381521224976, -0.7172703742980957, -1.968142032623291, 0.2744903862476349, -0.3962644636631012, 0.41324305534362793, 1.254657506942749, -0.4362369775772095, -0.3012729585170746, -0.9773755073547363, 0.4599398374557495, 0.635562002658844, 0.4574797749519348, 1.4889711141586304, 0.5665648579597473, 2.5827841758728027, 0.9158034324645996, 0.9616501927375793, 2.966714382171631, -1.0887224674224854, 0.459378182888031, -0.8733366131782532, -0.3747169077396393, 0.3473569452762604, -1.0887144804000854, 1.9688966274261475, -1.1371327638626099, 0.47457459568977356, 1.2330231666564941, 0.5412254929542542, 0.5172697901725769, 0.3931822180747986, -2.2749016284942627, -1.8801116943359375, 1.6741074323654175, 0.3221690058708191, 1.628124475479126, -1.154739260673523, -0.797467827796936, -0.8565952777862549, 1.0812873840332031, -0.7123952507972717, -1.9101860523223877, 1.2059239149093628, 0.16491562128067017, 1.1603010892868042, -1.0087823867797852, -1.0309226512908936, 0.48901239037513733, 0.1577303111553192, 0.19734863936901093, -0.9638895392417908, -0.5765681862831116, -0.3342004418373108, -0.6485891938209534, 0.7204585671424866, 0.16632984578609467, -0.6463712453842163, -0.5102779865264893, 0.7444448471069336, -0.2371624857187271, -1.2160025835037231, -0.362947016954422, -0.12665507197380066, -1.2923593521118164, -0.0385260134935379, 0.16405948996543884, 0.5789858102798462, -0.34284812211990356, -0.37088343501091003, -0.3245309293270111, 2.0128159523010254, -0.20672298967838287, 1.2481590509414673, 0.21247248351573944, 0.4004672169685364, 0.29899919033050537, -0.26992613077163696, -0.3383621573448181, 2.3913211822509766, -0.9430569410324097, -0.1264118105173111, -0.9422538876533508, 0.032656826078891754, -1.8541241884231567, -0.18677285313606262, 0.3897881507873535, 1.7559759616851807, -0.17560376226902008, 0.558264434337616, 1.878743052482605, 0.4285544455051422, 0.6721056699752808, -0.23777979612350464, -1.7356361150741577, 0.21180440485477448, 0.344892680644989, 2.482663869857788, 1.4805172681808472, 0.3075171709060669, 0.16041196882724762, -0.7461822032928467, 1.1639766693115234, 1.0053424835205078, 0.4591326117515564, 0.02426348626613617, -0.614890456199646, 1.4324402809143066, 1.1965302228927612, -0.6257408857345581, 1.2698851823806763, 1.2380810976028442, 1.195917010307312, -0.22520789504051208, -2.9692840576171875, -1.6633765697479248, 0.35091620683670044, 1.1825780868530273, -0.08029413223266602, -1.5255177021026611, -0.6506192088127136, 1.4324215650558472, 1.0753542184829712, 0.9966750741004944, -1.3768064975738525, -1.6090080738067627, -0.07856550812721252, -0.17004820704460144, -0.5397152304649353, 0.02828771248459816, -0.5782139897346497, -0.0374080054461956, -1.0635120868682861, -0.9136855602264404, 0.5968083143234253, 0.5315931439399719, -0.6837176084518433, -0.4782315194606781, 0.6950304508209229, -0.09775912016630173, 0.31860288977622986, -0.03128157928586006, -1.1773734092712402, -1.6014963388442993, -0.17653876543045044, 1.0641196966171265, -2.554633855819702, -1.275931477546692, 1.6794443130493164, -0.1801365166902542, -0.8548159599304199, -1.0925654172897339, -0.24932177364826202, -1.0636812448501587, -0.2401132732629776, 0.43981221318244934, -0.8232640027999878, 0.5171198844909668, 2.158142328262329, -0.07790669798851013, 0.32480311393737793, 0.17618310451507568, 0.3356281518936157, 0.22886890172958374, 1.4618326425552368, 0.11870378255844116, -0.6746528148651123, -0.19133861362934113, 0.11574654281139374, -2.0160410404205322, 1.0723822116851807, 1.848525881767273, -0.35271206498146057, 0.6977468729019165, 2.1956305503845215, -0.8516862392425537, 0.21650564670562744, 0.38029736280441284, 0.771858274936676, 0.6055321097373962, 1.3081406354904175, -0.22469154000282288, -0.7385256290435791, 1.9575129747390747, 1.294313907623291, 1.0233049392700195, 0.9504260420799255, 0.17530007660388947, 2.4307119846343994, -0.8677036166191101, -0.38139545917510986, -2.3987817764282227, 2.2545835971832275, 0.5863733887672424, -0.723331868648529, -0.3687581419944763, 1.9172897338867188, -1.1474205255508423, -0.30371183156967163, 0.04233633354306221, -0.3541755676269531, -0.12592025101184845, 0.8439428806304932, -0.7780008316040039, -0.5581232905387878, 0.5335035920143127, 0.4483727514743805, 1.0772160291671753, -0.3867303133010864, 0.8825465440750122, -1.358447790145874, -0.8764129281044006, 1.7286392450332642, -1.5393807888031006, -0.1234741285443306, -0.22148022055625916, 0.7262572050094604, -0.914845883846283, -0.7126644253730774, 1.1082289218902588, -0.3997792899608612, -0.03825696185231209, -0.08355922251939774, 0.45599833130836487, -0.12089843302965164, -0.49750134348869324, 0.10747280716896057, 1.240909457206726, 1.3929965496063232, -0.34747397899627686, 0.55828458070755, -0.5434610247612, 1.2419062852859497, 1.4590978622436523, 1.7594059705734253, -1.0897221565246582, 2.0267863273620605, -0.4552984833717346, -1.3161362409591675, 1.5658262968063354, 0.5254163146018982, 1.8120509386062622, 1.2430000305175781, -0.34632009267807007, -0.14953497052192688, -0.07250290364027023, 2.4979169368743896, -0.39681369066238403, 0.447035014629364, -0.20766569674015045, 1.250814437866211, 0.9731044173240662, -0.2906527817249298, -0.3503043055534363, -2.140155553817749, -0.06288626790046692, 1.144398808479309, 0.15268349647521973, -0.8106619119644165, 0.9577387571334839, 0.04544415324926376, 1.2998541593551636, -0.23131011426448822, -0.8816842436790466, -1.010509967803955, -0.8062374591827393, 0.029090123251080513, 0.9699819087982178, -1.0723779201507568, 0.21627219021320343, -0.3055417835712433, 1.1558526754379272, -1.3750154972076416, 0.4033162295818329, -2.1917927265167236, -0.5828045010566711, 0.2782115042209625, 0.6428569555282593, 0.5377488732337952, 1.3608269691467285, 0.15275630354881287, -0.3302743434906006, 0.1747758984565735, 0.06810285151004791, -1.4610708951950073, 0.46206361055374146, 0.03982594236731529, 0.5816054344177246, -0.3507256507873535, 1.866260290145874, -0.4570060968399048, 0.7472829222679138, -0.8289788365364075, 0.5241861939430237, 1.2717149257659912, -0.45509442687034607, -0.9010872840881348, 0.2547057271003723, 0.5484891533851624, -0.03492150083184242, 0.8873290419578552, -0.6773338317871094, 0.32873231172561646, 1.3137588500976562, 0.4149675667285919, -0.7368007898330688, -0.5171607136726379, -0.1395409256219864, 0.2967003881931305, 1.546785831451416, -0.8621209263801575, -0.2606427073478699, 0.01742762140929699, 0.4469403922557831, 1.0720300674438477, -0.5815941095352173, -0.4407438337802887, 0.1579352170228958, 1.8163923025131226, 0.515297532081604, 2.9138917922973633, 0.13629621267318726, 0.7264086008071899, -0.4558209776878357, 0.0892852172255516, 0.38866931200027466, 0.027944667264819145, 0.5058343410491943, 1.1787999868392944, 0.4035891592502594, -1.1770657300949097, -0.699160635471344, 0.33774036169052124, 0.4996764659881592, 1.3986958265304565, -0.5549142956733704, -1.3307992219924927, -1.0444598197937012, -1.2191540002822876, 0.5936859846115112, -1.112433910369873, -1.686660647392273, -0.6813716292381287, 0.054425496608018875, 0.4622921645641327, 0.9995176792144775, -0.0695384070277214, -1.219993233680725, 1.1801426410675049, 0.016899598762392998, -0.7720718383789062, -0.32804661989212036, -0.1255822330713272, -0.45343223214149475, 0.9474873542785645, -0.15929491817951202, 0.07133693993091583, -0.7739431262016296, -0.8545984029769897, -0.3061078190803528, 1.1267478466033936, 0.19530346989631653, -0.36091262102127075, -0.13957171142101288, -0.4890196919441223, -0.9319697618484497, 0.27849218249320984, 3.2888991832733154, 1.4189637899398804, 0.421062171459198, -0.9348209500312805, 0.3468851149082184, 0.9850670695304871, 0.9490557909011841, 0.2791809141635895, 0.09987680613994598, 1.3778406381607056, 0.5942663550376892, 0.0215599425137043, 1.6410332918167114, 0.7998949289321899, -0.772275984287262, 1.5039864778518677, -0.8672081828117371, 1.9135043621063232, 1.617400884628296, 1.3846584558486938, -0.8883554935455322, 0.9573037624359131, 0.24672651290893555, 0.5210198163986206, -1.0708664655685425, 1.2848094701766968, -1.867926001548767, 0.7517255544662476, -0.4842914044857025, 0.5298056602478027, 0.6078069806098938, 0.3045606017112732, -0.2756027579307556, 0.46817442774772644, 0.7250778675079346, 0.961015522480011, 1.0776695013046265, 1.4573497772216797, 0.16652196645736694, -3.01088547706604, 0.028955301269888878, 0.20894241333007812, -0.46162039041519165, 1.0891742706298828, 0.5802032947540283, -2.0237226486206055, -0.2650664448738098, -0.09042493999004364, 1.383245825767517, -0.9205355644226074, 0.12337958812713623, 0.1432025283575058, 0.26300278306007385, 0.22870196402072906, 1.9921061992645264, -0.9776713252067566, 0.8521570563316345, 0.9960127472877502, -0.714681088924408, -0.04472192004323006, 0.9246379137039185, 0.9710903763771057, 1.0518838167190552, -1.3898195028305054, 1.2779948711395264, -1.0943948030471802, -0.8558245301246643, -0.07261025160551071, -0.20927231013774872, 1.05863618850708, -1.5612510442733765, 0.24572305381298065, -0.8547565340995789, 0.44166916608810425, 0.5309023261070251, 0.029014721512794495, 1.7537938356399536, 1.2188624143600464, 0.014301028102636337, 0.23115293681621552, -1.799296498298645, -0.19794106483459473, 0.4485710561275482, -0.9755115509033203, -0.8339234590530396, -0.09471454471349716, -0.23193572461605072, -0.35678529739379883, -0.30247312784194946, 0.5737875699996948, -0.303842693567276, 0.7317981123924255, -0.6588424444198608, 0.42324405908584595, -1.6706886291503906, 0.5440444946289062, -0.6754134297370911, 0.16789181530475616, -0.5018788576126099, -0.8936629295349121, 1.185550570487976, 1.1951628923416138, -0.10235831886529922, 1.013250470161438, -0.9269413948059082, -0.6440281271934509, -1.4590340852737427, -0.017009327188134193, -0.628862738609314, 0.22658038139343262, 2.475011110305786, -0.6327065229415894, -0.24126216769218445, -1.7748827934265137, 0.26525962352752686, -0.526656448841095, -0.12075696885585785, 1.3340023756027222, 0.7212158441543579, -1.4653548002243042, 1.3132179975509644, -2.4146060943603516, 0.11411698907613754, 0.7313652038574219, 2.0557408332824707, -1.2833271026611328, -0.6347437500953674, -0.25655344128608704, 0.19026781618595123, -0.4546501636505127, 0.027223628014326096, 2.807983636856079, -0.6128873825073242, -0.943845272064209, 2.484487533569336, -0.908740222454071, -0.3916705846786499, -1.4794195890426636, 1.2312763929367065, -0.8821988701820374, -1.9146612882614136, 0.6499828696250916, -1.9553204774856567, -1.1796932220458984, 0.9086927771568298, -0.4281235337257385, -0.6256060600280762, -1.6143884658813477, 0.7233270406723022, -0.09098365157842636, 0.3373991549015045, -1.3560534715652466, 0.8557645678520203, -0.18436016142368317, 1.2351218461990356, -1.5645164251327515, -0.8263712525367737, 1.462860345840454, 1.1868501901626587, 2.048217535018921, 0.6430112719535828, 1.375914216041565, -0.7498283982276917, 0.9903492331504822, 0.7487462759017944, 0.1464608609676361, 0.03869360312819481, 0.8919726610183716, -4.183431625366211, -0.5438622236251831, -0.14268340170383453, -1.2423135042190552, -0.8141838312149048, -1.651125431060791, -0.9226033091545105, 0.05883859097957611, 1.9427566528320312, 0.01934661902487278, -0.6982600688934326, 0.054102227091789246, -0.847947359085083, 1.7133126258850098, 0.46749526262283325, 1.3656667470932007, -0.7373336553573608, -1.5816468000411987, -1.3788150548934937, -0.8322648406028748, -0.5284795165061951, -0.17231439054012299, -0.9656389355659485, -1.4970091581344604, 1.824057698249817, -0.6222085952758789, -0.03534979000687599, 0.44933876395225525, 0.629969596862793, -0.47057825326919556, 0.8508014678955078, 0.29244253039360046, -0.5461312532424927, -2.779801368713379, 0.814693808555603, -0.5677983164787292, 0.9834885597229004, 0.06391612440347672, 0.3266795873641968, 0.32622021436691284, -0.6673115491867065, -2.210625171661377, 0.8615981936454773, -0.49099069833755493, 0.8814343810081482, -0.04301295056939125, -0.20683714747428894, -0.9705954194068909, 0.5925449728965759, 0.04380112141370773, 0.27739378809928894, -0.5188137292861938, 0.8484799265861511, -1.6055705547332764, 0.04798075929284096, -0.7297239899635315, 0.19449292123317719, 1.9842195510864258, 0.5246126651763916, -1.3292741775512695, -0.013143952004611492, -0.041521526873111725, 0.9255954027175903, 0.32391107082366943, -0.24361974000930786], [-0.22611071169376373, -0.972758412361145, 1.3993487358093262, 0.8954334855079651, -0.949681282043457, 1.4851758480072021, -1.0271185636520386, -0.08504123240709305, -0.15456104278564453, -0.16777555644512177, -1.5184283256530762, -0.15044613182544708, 0.9463807940483093, 0.732085645198822, 2.3232479095458984, -1.3805437088012695, -0.1254236102104187, 0.48413151502609253, 1.170322299003601, -0.012735561467707157, -0.10939116775989532, -0.5097293257713318, 1.8308112621307373, -0.12987928092479706, -0.8828309178352356, -1.1197659969329834, 1.2065563201904297, 0.5080297589302063, -1.3457660675048828, 1.5951318740844727, 0.4197436571121216, 0.33569619059562683, -0.984515905380249, -1.0492520332336426, 0.820727527141571, -0.5352823734283447, -0.651497483253479, 1.033140778541565, -0.48091283440589905, 0.2884838283061981, 1.0198813676834106, 0.5975759029388428, 1.1659784317016602, 0.91251540184021, 0.2059953361749649, 0.0645674392580986, 1.0706207752227783, -0.4330102801322937, 0.7579762935638428, -0.3191705644130707, 0.36197715997695923, -1.7871372699737549, -1.0852638483047485, -0.98536616563797, -0.8034572601318359, 1.7520382404327393, 2.1452460289001465, -1.1839485168457031, 0.1686091274023056, -1.4951082468032837, -0.8177816271781921, 0.9855679273605347, 1.6167395114898682, -0.0012270688312128186, -0.8842662572860718, -0.5165368318557739, 0.4758314788341522, -0.5095682740211487, 0.35153552889823914, -1.0920966863632202, 0.7944191694259644, -0.3064579963684082, 0.3501320779323578, 2.876716375350952, 1.452144980430603, 0.6042402982711792, 0.4293292164802551, -0.4025645852088928, -0.1491725742816925, -0.15708224475383759, 1.8647040128707886, 1.3801766633987427, 0.23896946012973785, 1.4084408283233643, 1.0093698501586914, 1.183548092842102, 1.14682137966156, 0.16078117489814758, 0.04608059674501419, 0.4604703485965729, -0.1146293580532074, -1.2280793190002441, -0.4512270390987396, -1.4045860767364502, 1.5441957712173462, 0.4056374430656433, 0.11494426429271698, 0.9496508240699768, -1.326522946357727, 0.5779141783714294, -0.006879315711557865, -1.0873353481292725, 0.675991952419281, 0.37210267782211304, 0.03367657959461212, -0.6735295057296753, 1.5637468099594116, 1.0824638605117798, -0.1042979508638382, 2.503657579421997, 0.4659074544906616, 0.5257478952407837, 0.2086355984210968, 0.5320135354995728, -0.09093073010444641, -0.22167883813381195, 0.008737909607589245, 0.9127618670463562, -0.597608745098114, 1.1987760066986084, -0.1527056246995926, -0.44665929675102234, -2.783729076385498, 1.8606808185577393, 0.46724891662597656, 0.7908505797386169, -2.050257682800293, -2.166311502456665, -0.34636953473091125, -0.5989989042282104, -0.7534518241882324, -1.4609029293060303, -0.5111345052719116, 1.4508105516433716, 2.494814395904541, 2.082164764404297, 0.09773016721010208, 0.4558701813220978, 0.38804757595062256, -0.2650551199913025, 1.00369393825531, 0.6618953347206116, 0.8424234390258789, -0.509654700756073, -0.35898080468177795, -1.2298487424850464, 0.25758621096611023, -0.9901198744773865, -1.0006933212280273, -0.8149951696395874, -0.929600179195404, 0.3327251970767975, 0.6998111605644226, 0.13711977005004883, -1.093938946723938, -1.8763542175292969, -0.7904746532440186, 0.6661428213119507, -1.639406442642212, -0.8994614481925964, 1.2760279178619385, 0.46631482243537903, 1.3290009498596191, -0.028300555422902107, 0.6454765200614929, -0.10524684190750122, -0.5970712304115295, -0.6577278971672058, -0.16105172038078308, -1.273522138595581, 0.8032184839248657, 0.05091528594493866, 0.03724223002791405, -0.4443749487400055, 0.26981058716773987, -0.5936780571937561, -0.7099090814590454, -2.119706392288208, -0.27516940236091614, -2.042750358581543, 2.6616439819335938, -0.5009725093841553, 0.8157792687416077, -1.5476837158203125, 0.1377825140953064, 0.038789018988609314, -1.260467290878296, 0.25072768330574036, 0.9563094973564148, 1.3202250003814697, -0.3785417079925537, -0.2545824944972992, 2.6617376804351807, 2.2845587730407715, -1.8551937341690063, 0.27480605244636536, 0.7921104431152344, -2.201007604598999, 0.5992682576179504, -0.6294308304786682, -0.36628371477127075, -1.4149820804595947, 0.5949693918228149, -1.236243486404419, 0.350349485874176, 0.20711535215377808, -0.0839756429195404, 1.251063585281372, 0.2139119952917099, 0.06616826355457306, 0.6674582362174988, -1.6641727685928345, -0.36794471740722656, -1.345100998878479, -3.1951327323913574, -1.4008067846298218, 1.3781782388687134, -0.022154293954372406, -1.7175925970077515, -2.40346360206604, -1.086452841758728, 1.279658317565918, -2.4068868160247803, -0.6468768119812012, -0.2008584588766098, -0.5882791876792908, -1.3073688745498657, 0.8260207772254944, -1.0727436542510986, 0.08736921846866608, 0.5551586151123047, 0.18270370364189148, -1.2170706987380981, -0.523303747177124, 0.9438757300376892, -0.0986374020576477, 1.3015942573547363, 0.7479258179664612, 0.7639641761779785, -0.7060481309890747, -0.05836377292871475, 1.0253630876541138, 0.6651415228843689, -0.5794309377670288, 0.5745768547058105, -0.29209914803504944, -1.777398705482483, -0.6901417970657349, -1.275938868522644, 0.10574888437986374, 1.0675822496414185, -0.35321158170700073, -0.20138272643089294, 0.6837954521179199, -0.3319690227508545, -1.9296208620071411, 0.8737702369689941, 0.3753194212913513, 0.0761169046163559, 1.3643767833709717, 2.285139560699463, -0.27113714814186096, 0.8136076927185059, -0.005361528135836124, -1.4835928678512573, 0.9299226999282837, -0.0978844165802002, 0.7061933279037476, -1.8442894220352173, -0.38996827602386475, 0.8986557126045227, 1.8728214502334595, 0.9712364673614502, -0.8035322427749634, 0.698896050453186, 0.964669942855835, 0.21807043254375458, -0.44686076045036316, -0.10729119926691055, 1.1551138162612915, -1.0204731225967407, -0.7254839539527893, 1.972929835319519, 0.017559565603733063, 0.5480262041091919, -1.2817660570144653, 1.1982554197311401, 0.6937042474746704, -0.8101075887680054, 0.36262422800064087, 0.2508997917175293, -1.6045811176300049, -0.29255661368370056, 0.9472665786743164, -0.8537537455558777, -1.6376514434814453, 1.5565599203109741, -1.6278109550476074, -0.16961975395679474, 0.2264195680618286, 0.20589318871498108, -0.638914942741394, 1.3115829229354858, 0.1327977478504181, -0.7812873721122742, -0.16343429684638977, 1.02764093875885, 2.083129405975342, -1.5696570873260498, -1.0416672229766846, -0.1612415462732315, -0.1012311652302742, -0.8392553329467773, 1.6700712442398071, -0.4350791573524475, 1.2127821445465088, -1.071581482887268, -0.11833418160676956, -0.20496830344200134, -0.8203713297843933, 2.256559371948242, -0.4244552254676819, -1.4023747444152832, 1.7034416198730469, 1.5824034214019775, 0.4512988328933716, -1.4662009477615356, -1.1103746891021729, -0.6513460874557495, 0.16216257214546204, 0.35219088196754456, 1.23558509349823, -0.9817150235176086, -0.06281198561191559, 0.5998531579971313, -2.3238213062286377, 1.6076124906539917, -1.249875545501709, 1.2452512979507446, -0.5610148906707764, -0.5571937561035156, 0.23630225658416748, 0.5919380784034729, -0.12275299429893494, -0.1372651904821396, -0.7225314378738403, 0.4424968659877777, -0.9848203659057617, 2.2488157749176025, 0.2067185342311859, 0.01600399799644947, 0.06630779802799225, -2.586613178253174, 0.3452761769294739, 0.07861300557851791, 0.25496217608451843, 0.2360367327928543, -1.5004369020462036, 0.04621627554297447, -0.8857045769691467, 1.1152217388153076, 0.5881803035736084, -0.14506423473358154, -0.16417454183101654, 0.4407942295074463, -1.4168298244476318, -0.8898941278457642, -0.5737497806549072, -0.34295734763145447, 0.6899561285972595, -0.34355539083480835, 1.39457368850708, 0.8591631650924683, -1.5956270694732666, -0.3933486342430115, 0.38174453377723694, 0.6797799468040466, -2.1226675510406494, 1.135599136352539, -1.4257042407989502, -0.07310041785240173, -0.6329177021980286, 0.05374831333756447, -0.0978466123342514, -0.2026871144771576, 0.2510680854320526, 1.1856977939605713, 0.26907065510749817, -0.41746851801872253, -0.18544802069664001, -1.3775389194488525, -0.82793790102005, 0.2451830506324768, 1.9688804149627686, 0.012806595303118229, 0.1588975489139557, 1.6164841651916504, -0.8686107993125916, 0.5793426632881165, -3.01396107673645, 0.1452220231294632, -0.11355923861265182, 1.1803478002548218, -0.2543058395385742, -0.36750590801239014, -0.4237193465232849, -0.0010242409771308303, 0.8455490469932556, 0.20767353475093842, 0.4909810423851013, 0.21309539675712585, -1.4406837224960327, -1.4640016555786133, -0.06789911538362503, 3.3276495933532715, -1.1201796531677246, -0.3558084964752197, -0.8494624495506287, 1.7401710748672485, -1.259848713874817, -0.6051615476608276, 0.943382203578949, -0.6660849452018738, -0.28391310572624207, -1.0631197690963745, 0.13069456815719604, 1.2003827095031738, 0.593076765537262, -1.449676513671875, 1.5595186948776245, -1.5647896528244019, 0.2991507649421692, 0.9276291728019714, -0.7820238471031189, 1.089955449104309, 1.0191214084625244, -0.6414435505867004, 1.0084567070007324, -0.2090274840593338, -0.7165332436561584, 0.8400900959968567, -0.3766583800315857, 0.3688623905181885, 0.4730478823184967, 0.26651182770729065, -1.5812455415725708, 0.3295576870441437, 1.045032262802124, -2.1753623485565186, -1.0852694511413574, 0.2712174355983734, 0.6513383388519287, 0.596262514591217, -1.0306401252746582, 0.273691326379776, 0.09853582829236984, 1.138159990310669, -1.096269965171814, -0.9782804846763611, -0.33529889583587646, -0.49503225088119507, 1.110453724861145, -0.2741667628288269, -1.0665823221206665, -0.6502170562744141, 0.8397599458694458, 0.27920275926589966, -0.2597602605819702, -0.7127761244773865, -1.9007271528244019, 1.1163235902786255, -1.097585916519165, 0.015695560723543167, 0.2070494294166565, 0.5624681115150452, 0.03658909350633621, -0.39558345079421997, -0.2110559344291687, -0.30372166633605957, 1.860330581665039, 0.6882044076919556, -0.3824881911277771, 0.6035832762718201, -1.0424834489822388, 1.2048227787017822, -0.5601915717124939, 0.30130618810653687, -0.7739536762237549, 0.07843202352523804, -1.6292619705200195, 0.6538318991661072, -0.3394165337085724, -1.2408385276794434, 0.5776069760322571, 0.4863602817058563, 0.04479004442691803, -0.49601879715919495, 0.8715077638626099, -0.4113205373287201, -2.0443122386932373, -0.30245742201805115, 1.0514918565750122, 0.8007027506828308, -0.09536226093769073, 0.6186273694038391, -1.108890414237976, 0.8734024167060852, 0.20494917035102844, 0.5242298245429993, -0.9612396359443665, -2.602177143096924, 0.14257539808750153, 0.9679425358772278, 1.298362374305725, 0.20623034238815308, 1.1969671249389648, -1.4294533729553223, -0.028541112318634987, -0.20562289655208588, -0.8874239325523376, -1.1114997863769531, 0.44389447569847107, 0.8767840266227722, 0.25044870376586914, -0.7881852984428406, -0.318340927362442, -0.9016708135604858, 0.6819615960121155, -1.7450332641601562, -0.13119740784168243, 0.22944562137126923, -0.504555344581604, -1.5875976085662842, -0.35784584283828735, 2.227396011352539, -1.6926320791244507, -0.42945927381515503, -2.0745038986206055, -0.8427858948707581, -0.7010995149612427, -1.0987399816513062, -2.2706148624420166, 0.9859173893928528, -0.8929553031921387, -0.7401930093765259, -0.6017031073570251, 0.8887810707092285, -0.5936979055404663, 1.083984136581421, 0.22205525636672974, 0.47106391191482544, 0.15133927762508392, 1.3249483108520508, 1.197584629058838, -0.38331669569015503, 0.5795876979827881, -1.0495882034301758, -1.5525785684585571, 0.6192982196807861, 0.7742892503738403, -0.27008968591690063, 0.2991638779640198, 0.2200169414281845, 1.3768808841705322, -1.0221694707870483, 0.2515435814857483, 1.2211745977401733, 0.03792451694607735, 0.4688780605792999, 0.32103094458580017, -0.1807228922843933, 0.9120608568191528, 0.051352228969335556, -1.1634000539779663, -0.6515810489654541, 1.0094255208969116, 1.773006558418274, -0.4951897859573364, -0.09921975433826447, -2.2711944580078125, -0.17562681436538696, -0.679194986820221, 0.4340314269065857, -1.0590448379516602, -0.9350277781486511, -0.1882394254207611, 0.7969110608100891, 0.5498683452606201, 0.11787495017051697, 0.6896185874938965, -1.5221222639083862, -0.2189861238002777, -0.6585209965705872, -0.28201568126678467, 0.2541101574897766, -0.074317067861557, -0.1827792227268219, -1.1859960556030273, 0.6217462420463562, -1.806169867515564, -0.5300478339195251, 0.8881046772003174, -1.8617968559265137, -1.2986353635787964, 1.1805964708328247, -0.7288558483123779, 0.029280198737978935, 0.7984399795532227, -1.4540932178497314, -0.24769821763038635, 0.9372186064720154, 0.24779002368450165, -0.39587146043777466, 1.3080042600631714, 0.44891729950904846, 1.081246018409729, 1.2625138759613037, 1.4979965686798096, -0.4360087215900421, -0.5384560823440552, 0.7778126001358032, -1.2544019222259521, 0.9202086329460144, 0.832209050655365, -0.4948897659778595, 0.29728201031684875, -0.6685219407081604, 0.2254239022731781, 0.13147149980068207, -1.454686164855957, -0.4872877597808838, -0.12241821736097336, -0.5131959915161133, -0.040910664945840836, 1.3120331764221191, 0.0819609984755516, -0.9384113550186157, 1.4852843284606934, 0.853010356426239, -0.021289488300681114, 0.26516756415367126, 0.536943793296814, -1.0187844038009644, -0.999339759349823, -1.1839300394058228, 0.22988101840019226, 0.014422631822526455, 1.6083216667175293, 0.050267212092876434, -0.9105170965194702, 0.9245023131370544, -1.6173148155212402, 1.338875412940979, -0.31136181950569153, -0.25167015194892883, -0.1828664243221283, -0.6177293062210083, 2.6120119094848633, 1.1331266164779663, -0.03979130834341049, -0.04840967804193497, -0.45268476009368896, 0.02426527999341488, -0.46741169691085815, 0.4724079668521881, 0.7478504180908203, -1.1653225421905518, 0.6716165542602539, 0.27478092908859253, 0.41609808802604675, 1.849743366241455, 0.7771880626678467, -1.3361073732376099, -0.19248457252979279, -1.0262360572814941, -0.1429976224899292, -0.10608804225921631, -1.3879703283309937, 0.03898399695754051, 0.15813618898391724, -1.0046688318252563, 1.2095377445220947, 0.4422330856323242, -0.32341307401657104, -2.238596200942993, -0.37737134099006653, -0.11998540163040161, -0.4548356235027313, 0.4124491810798645, -1.0021768808364868, 0.7264750003814697, 0.017448334023356438, -1.4229110479354858, -0.32532623410224915, -0.7414954900741577, 0.5649678707122803, -0.6051259636878967, 0.2115701138973236, 2.6335158348083496, -1.7405494451522827, 1.042974829673767, -0.8920584321022034, -1.2172882556915283, 0.26700201630592346, 0.18814727663993835, -0.06295936554670334, 1.0665401220321655, 0.18926483392715454, 0.5191702246665955, -0.816506028175354, -1.025273323059082, 0.17341330647468567, 0.7490069270133972, 1.5301623344421387, -1.4459174871444702, -1.7662279605865479, -1.1431927680969238, 0.5680530071258545, -0.38554930686950684, 1.487779140472412, 0.38906899094581604, 1.010676622390747, 0.23564474284648895, -0.04673141986131668, -2.036351442337036, -0.3306320607662201, -0.71341472864151, 0.045185890048742294, -0.7809299230575562, -0.18891297280788422, -1.9657707214355469, -1.2537448406219482, 0.5632998943328857, 0.5194131731987, 0.04456110671162605, 0.9999645948410034, -0.674328625202179, 1.1442445516586304, 1.2788424491882324, 0.1520935297012329, 0.8743733763694763, -1.7554097175598145, 1.0721989870071411, 0.7188623547554016, 1.1313508749008179, -0.6401631832122803, 0.0511077344417572, 0.8330367207527161, 1.4434576034545898, 0.2777334451675415, 1.211116909980774, -0.5486890077590942, 1.0244450569152832, -0.8567304611206055, -1.3188457489013672, -1.0187724828720093, 1.7615982294082642, 1.1216511726379395, 0.5297643542289734, -0.14015880227088928, 0.9369105696678162, 0.7044406533241272, -2.2758941650390625, 0.580022394657135, -0.5507971048355103, -0.854591429233551, -0.03824058920145035, 0.9998921155929565, 1.7135627269744873, -1.21819007396698, 1.069048523902893, -0.581842303276062, -0.4828873574733734, 0.22510598599910736, 0.5843958258628845, -0.6028570532798767, 0.31165334582328796, 0.28649675846099854, -1.350465178489685, -0.05170001462101936, 1.5604956150054932, 1.82862389087677, 0.4418691396713257, 1.1790519952774048, -0.17470000684261322, -2.393009662628174, -1.7946456670761108, 1.5055915117263794, 0.3444649875164032, -0.5881736874580383, -1.2092357873916626, -0.04025275632739067, 1.0772697925567627, 0.6654406785964966, -1.5552473068237305, 0.5708165168762207, 0.5173537731170654, -1.6356561183929443, 0.34694385528564453, -0.7379279136657715, 1.2371779680252075, 0.42295584082603455, 1.5025687217712402, 1.6893638372421265, 0.3063468635082245, 1.5003840923309326, -1.899943470954895, 0.5622697472572327, -0.5025186538696289, 0.656299352645874, 0.8034893870353699, -0.7751300930976868, 0.8087711334228516, -1.1920366287231445, -0.012074032798409462, -0.20750318467617035, 0.3853473663330078, 0.9588828086853027, -0.23318906128406525, 1.583310842514038, 0.6511693000793457, -0.5970165729522705, 1.0010104179382324, -2.4676313400268555, 0.3817984163761139, 1.685880422592163, -1.2178704738616943, 0.4677233099937439, 0.8380081653594971, -1.5716025829315186, -0.3322067856788635, 0.6359710693359375, -0.29704785346984863, -1.0918233394622803, 0.5307093262672424, -1.1497430801391602, -0.6924824118614197, 1.870058536529541, 2.1846089363098145, 0.35710760951042175, 0.7836652994155884, -1.7320749759674072, -0.5871980786323547, 0.7830672264099121, 0.7171014547348022, 0.9061485528945923, -0.7767636179924011, 0.08745121210813522, 0.6370162963867188, 0.2504432201385498, 0.36037567257881165, -2.779493808746338, 0.4485688805580139, -0.1560392677783966, -1.3471192121505737, 0.3695790767669678, 0.09378678351640701, -0.3662995994091034, 1.080329179763794, -0.4162140488624573, 0.0385090596973896, 0.409707248210907, 1.411661148071289, -0.0871523916721344, -0.7724316716194153, 0.5666872262954712, 2.0668859481811523, -0.5781705379486084, -0.44120240211486816, 0.07279585301876068, 0.08865640312433243, -1.5615657567977905, -1.0942472219467163, -0.2655586302280426, -0.3949604332447052, -0.08692413568496704, -1.0349808931350708, -1.1928956508636475, 0.8410865664482117, 1.1133791208267212, -0.5943663716316223, 0.09795740246772766, 0.6242731809616089, -0.14825496077537537, 1.0809135437011719, 1.1311455965042114, 0.30848342180252075, 0.21589410305023193, 0.056442346423864365, -1.9777154922485352, -0.7431287169456482, -0.5645994544029236, 0.035954929888248444, -0.6843947172164917, -1.3987518548965454, 1.1544761657714844, 0.26027384400367737, 1.0578885078430176, 1.2800498008728027, -0.661749005317688, 0.5061050057411194, -3.3967528343200684, -1.316973090171814, -0.030086886137723923, -1.0779870748519897, -0.7512893676757812, -0.5394865274429321, 0.8480337858200073, -0.6262285709381104, 0.3413551449775696, 0.9706153869628906, -1.6967424154281616, -0.32323890924453735, -1.010606050491333, -1.9358255863189697, 0.01715453527867794, 0.49365854263305664, -2.1098368167877197, -0.7721918821334839, 0.823683500289917, -1.0561118125915527, -0.31369438767433167, 0.40053534507751465, -0.5455631613731384, -0.8279470205307007, 1.7767550945281982, 0.22620977461338043, -0.4216497242450714, -0.14998069405555725, -1.2156898975372314, -1.075254201889038, 0.007050820626318455, 3.2281785011291504, 0.2956662178039551, 1.2147799730300903, 1.301020622253418, -0.9786892533302307, 1.9745826721191406, 0.5385497212409973, 0.5165026783943176, 2.111699342727661, 0.6344125866889954, 0.871782124042511, -0.540266752243042, -1.128341794013977, 0.12619225680828094, 0.14377623796463013, 1.3709039688110352, 1.0099873542785645, 1.5256413221359253, -0.9290363788604736, 0.5546350479125977, 0.08412273228168488, -0.14034409821033478, -0.748253345489502, 0.46686434745788574, -2.086066246032715, 0.10149750858545303, -0.13115204870700836, 0.41769352555274963, -0.598145604133606, 0.582314133644104, 0.5354400873184204, -2.3104732036590576, -0.08992313593626022, 2.168337821960449, -0.2252180427312851, -0.6710516810417175, 0.507870614528656, -0.4505373239517212, -0.20495086908340454, 0.39558836817741394, 0.957486093044281, -1.953329086303711, 1.4926525354385376, -0.9955644607543945, 0.8447805643081665, -1.252974510192871, -0.18209969997406006, -0.17024125158786774, 0.6364908218383789, 1.1326911449432373, 0.6292442679405212, -0.8272446393966675, 1.0083801746368408, -0.36712929606437683, -0.6324232816696167, 0.16665118932724, -0.37307295203208923, -2.0087435245513916, 0.14627718925476074, -0.8877594470977783, -1.4861643314361572, 2.3764190673828125, 0.484123557806015, -0.007141007576137781, 0.19325019419193268, -1.1083152294158936, 1.2477517127990723, 1.5548276901245117], [1.2938637733459473, -0.1737719178199768, 0.7611650228500366, -0.1887471228837967, 0.7578172087669373, -0.7347404360771179, 1.3490113019943237, 0.8724361062049866, 0.6927900314331055, -0.7580758929252625, 0.9100019335746765, -1.7069096565246582, -0.9433671832084656, 1.9957689046859741, -0.4556390941143036, -1.8280868530273438, 0.19547386467456818, -0.7021582722663879, -1.8611291646957397, -0.1523282825946808, -0.15611396729946136, -2.170311212539673, -0.12404492497444153, -1.6166889667510986, -1.9855340719223022, -1.221140742301941, 0.772408127784729, 1.7031080722808838, -2.1161110401153564, 1.5722390413284302, 0.20795628428459167, 0.6648416519165039, -1.3234726190567017, 0.7022198438644409, -1.967435359954834, -0.09670531749725342, 0.08480194211006165, 0.31993040442466736, 1.0019632577896118, -0.644905149936676, -1.6331239938735962, -1.6444051265716553, -0.4227032959461212, 0.9693530797958374, 0.5864170789718628, 0.3602818250656128, -0.07231365889310837, 1.444534182548523, 1.2369580268859863, 1.946138858795166, 1.5521129369735718, 0.8542713522911072, -0.5418572425842285, 0.8625733256340027, -0.9844968914985657, 0.01814240776002407, -1.5192806720733643, 1.647984266281128, -0.5876110792160034, -0.61399245262146, 0.0077578299678862095, -1.5503754615783691, 1.3895912170410156, -1.6977646350860596, -0.6031211018562317, -0.5072579979896545, 0.03480897843837738, -0.586827278137207, -1.0113918781280518, -0.6065108776092529, 0.5477116107940674, 0.5534188747406006, -0.24610036611557007, 0.6955652832984924, -1.3548535108566284, -1.488498330116272, 1.1291135549545288, 2.6909053325653076, -1.400047779083252, -0.646488606929779, 0.2963896095752716, 1.199472427368164, 0.868748664855957, -2.1481761932373047, 0.6833810806274414, -2.0563063621520996, -0.5247704982757568, -0.7665644884109497, 0.6375775337219238, 0.4660261869430542, -0.8626066446304321, -0.4499443471431732, -1.336455225944519, 0.7878115773200989, 0.21216754615306854, -1.6563615798950195, 0.5021986365318298, -0.06338851153850555, -0.37927529215812683, -1.031812071800232, 0.43786537647247314, -1.6688798666000366, 0.9219298362731934, 0.15517136454582214, -0.1890178918838501, 0.19885388016700745, 0.8246151208877563, -1.166078805923462, -0.11691612005233765, 0.0886407271027565, 0.009659664705395699, -0.22588546574115753, -0.19042851030826569, -0.03511805459856987, -0.5156245231628418, -1.0790773630142212, -0.5485257506370544, -2.3035528659820557, -0.4486580193042755, -1.349280595779419, 0.6411260962486267, 0.17775298655033112, -0.31964966654777527, 0.9156391024589539, 0.6984301805496216, 0.2233830690383911, -0.034906551241874695, 0.9138246774673462, -0.5966286063194275, -0.01885043829679489, 0.43297356367111206, 0.46972212195396423, 1.3801223039627075, 0.5218639969825745, 0.9130925536155701, 0.6586564183235168, -1.0031874179840088, 1.1884093284606934, 1.3092111349105835, -1.7623255252838135, -1.0734957456588745, 0.1741190254688263, 0.07356058061122894, -1.0368252992630005, 0.07981663197278976, 1.1522855758666992, 0.30413487553596497, 1.647294044494629, 0.5470528602600098, 0.4707856774330139, 0.6421308517456055, -0.23949739336967468, -0.1242590844631195, -0.471870094537735, -0.05979045107960701, -0.29125088453292847, -1.2274444103240967, -0.9000808596611023, 0.13388676941394806, -0.14541880786418915, -1.1440455913543701, -0.6467487215995789, -1.0152086019515991, 0.7365949153900146, -0.9416397213935852, 0.264958918094635, 0.9448526501655579, 0.42641791701316833, 0.1199611946940422, -0.94439297914505, -0.174638569355011, 0.3952081501483917, -0.3163398504257202, -0.522456705570221, 0.01382363960146904, 0.37988731265068054, 0.1437753438949585, 1.1423242092132568, 0.3693751394748688, -0.24556240439414978, 0.8584308624267578, -0.09296537190675735, 0.1807638555765152, -1.3655575513839722, -0.7368886470794678, 0.4694978892803192, 0.5630563497543335, 1.0404025316238403, -1.956063151359558, 0.2023695707321167, -0.8759958148002625, 2.4573237895965576, -0.6490933299064636, -0.24134837090969086, 0.23296073079109192, -0.049440138041973114, -0.6019477844238281, -1.7714029550552368, -0.08512500673532486, 0.6236260533332825, -0.28030523657798767, -2.0580008029937744, 0.58326655626297, -0.0875108540058136, -0.6032309532165527, 1.8388515710830688, -1.4186066389083862, 2.1061484813690186, -0.3318137526512146, -0.7233409881591797, -2.207611322402954, 0.8155540227890015, 0.4755544364452362, -1.7750619649887085, -0.5304338932037354, -0.2665041387081146, 1.7212685346603394, 0.30881696939468384, -0.5966644883155823, 1.131736397743225, 2.185565948486328, 0.9697645306587219, 2.203646659851074, 1.3124561309814453, 0.12669114768505096, -0.732555627822876, -0.07192058116197586, -1.3764394521713257, 1.4436371326446533, 0.8951073884963989, -1.0049601793289185, 0.18256860971450806, 0.7877227663993835, 1.0172691345214844, -0.3539186120033264, -1.2129825353622437, 0.9437254071235657, -0.9161590337753296, -1.8463932275772095, -1.2109827995300293, -0.7696347236633301, -0.31094661355018616, -0.6725435853004456, 0.0019395590061321855, 2.8334920406341553, -0.43950873613357544, 0.3069525361061096, 1.3513633012771606, -0.45172637701034546, -0.5268642902374268, -0.21691834926605225, 0.49710381031036377, 0.19611355662345886, 0.6803886890411377, 0.9069650173187256, -0.08614102751016617, 0.6524441242218018, -0.23026330769062042, -0.02939421311020851, 0.633809506893158, 0.47111082077026367, 0.08446186035871506, 0.5770272016525269, 0.7346891164779663, -0.1406719833612442, 0.8226205110549927, 0.44974711537361145, 0.6852574944496155, 1.0344332456588745, -0.9757714867591858, 0.5855000615119934, -1.128244400024414, 1.2425872087478638, -1.0784289836883545, 1.16805100440979, 0.5326927304267883, 0.7008334994316101, 1.025162696838379, -0.35860154032707214, -1.0588690042495728, -0.5226198434829712, 0.2347622811794281, 0.8724572658538818, 0.6060154438018799, -0.6547166705131531, 1.4551329612731934, -0.6902835369110107, -2.7982962131500244, -0.8905680179595947, 1.6270548105239868, 0.5436350703239441, -0.25732213258743286, -0.07071094959974289, -2.1163675785064697, 1.5456562042236328, -0.7316917181015015, 0.07980222254991531, 0.6556603908538818, 0.524438202381134, 0.007754516322165728, -0.5730524063110352, 0.8563124537467957, 1.419299840927124, -0.2648199796676636, -0.24950793385505676, -0.590487003326416, 0.13282617926597595, 0.7699095010757446, -0.23430827260017395, -0.46939975023269653, -0.9476884007453918, -0.5695288181304932, -1.3962877988815308, -1.4891445636749268, -0.9656259417533875, -0.14658001065254211, -0.2752120792865753, 0.18240845203399658, 0.7477962374687195, 1.2780967950820923, 0.23816539347171783, 0.24823743104934692, 0.8700118660926819, -1.4674444198608398, -0.3940131366252899, 1.1687015295028687, 0.4678211212158203, 0.33949804306030273, 0.318936824798584, 0.17429198324680328, -3.0606930255889893, -0.14095498621463776, 0.18856431543827057, 0.22252719104290009, 0.052225250750780106, -1.1888490915298462, -0.8541401028633118, 0.0957370474934578, 0.23715683817863464, -0.36300188302993774, -0.5106130838394165, 2.1958870887756348, -1.1169694662094116, 1.7984164953231812, -1.1912771463394165, -1.410019040107727, 1.3797720670700073, -1.4247454404830933, -1.0346013307571411, -0.697289228439331, -0.09649982303380966, -0.16755962371826172, 1.019426941871643, -0.4356614351272583, 0.28609853982925415, 1.3261862993240356, 0.12505051493644714, -0.020227542147040367, 0.43980056047439575, 0.09971689432859421, 0.4943446218967438, -0.16580165922641754, 1.6206766366958618, 0.20821283757686615, 0.4356856644153595, -0.1791016012430191, 0.31088805198669434, -0.6935256719589233, -1.2381643056869507, 0.40084177255630493, 2.341417074203491, 0.02894926816225052, -1.083568811416626, -0.39292091131210327, -0.32822850346565247, -1.4175527095794678, -0.9576748013496399, -0.9779530167579651, -0.10695423185825348, -0.4187607169151306, 0.007412449456751347, -0.927260160446167, 1.4522888660430908, 0.7934343218803406, -1.2917115688323975, -0.26361286640167236, -0.016747374087572098, -0.18514646589756012, 1.2014410495758057, -0.6686475276947021, -0.6431975960731506, -0.45904532074928284, 0.5433061718940735, 0.15479901432991028, 0.808777391910553, 1.978860855102539, -0.7932290434837341, 0.6789907217025757, 1.0682429075241089, -2.007718324661255, 0.7110463976860046, -0.5002439618110657, 0.26840105652809143, 0.5179978013038635, 0.9927492737770081, 0.6368340253829956, -1.7899891138076782, -0.8145284652709961, -0.20899567008018494, 0.24552638828754425, -0.24907591938972473, -1.552466869354248, 0.5460392832756042, -0.03714553266763687, -0.7375627756118774, -0.7747921943664551, -0.7212917804718018, -0.7374034523963928, 2.4546425342559814, -0.33921074867248535, 1.6469026803970337, -0.8344932198524475, -0.2747208774089813, -1.0045284032821655, -0.9853926301002502, -0.006883061956614256, 0.6620866060256958, 0.7033328413963318, 0.26810136437416077, -0.5409693717956543, 0.49695339798927307, -1.3264278173446655, -0.7618439793586731, 0.3884172737598419, -1.034935712814331, 0.22952136397361755, -0.5333251953125, 2.3006114959716797, 1.4525139331817627, 1.6328918933868408, -1.3870899677276611, 2.1953132152557373, 0.4544326961040497, 0.41108766198158264, -0.9119791388511658, -1.0714695453643799, 0.7538876533508301, 1.2997301816940308, 0.29643499851226807, 1.2115260362625122, -0.24267475306987762, 0.8665846586227417, -1.582069993019104, 0.04371322691440582, -0.28149938583374023, -0.8960721492767334, 2.118072986602783, -1.5742305517196655, 0.3348122537136078, -1.1620495319366455, 0.2217906266450882, 0.03535478189587593, -1.6453882455825806, 0.589478075504303, -1.135719895362854, -0.6195263862609863, 0.6523464918136597, 0.1532960832118988, -0.06807157397270203, 0.4195161461830139, -0.45201295614242554, 0.5745370388031006, 0.2258468121290207, -2.5891401767730713, -0.16459542512893677, -1.4330945014953613, 0.3355529010295868, -0.6812314391136169, -0.23804856836795807, -2.5159800052642822, 0.5737864375114441, 0.2583378553390503, -0.40056899189949036, 0.45520472526550293, 0.1223134994506836, -1.1849675178527832, -0.423796683549881, -0.8541079759597778, -0.9360129237174988, -0.5011366605758667, 0.9422675371170044, 1.520219326019287, -1.010822057723999, 0.08530832827091217, 0.2833133339881897, 0.4576062262058258, 0.05358891934156418, -0.15956273674964905, 1.008972406387329, 0.31540751457214355, -0.032585423439741135, 0.9489980936050415, 1.5253934860229492, 0.354157954454422, -0.8095256686210632, 1.2609496116638184, 0.2189774215221405, 0.4186994731426239, -0.6168140769004822, 0.4268780052661896, -1.4736809730529785, 0.39450815320014954, -0.02922406606376171, 0.6921079158782959, -0.26749828457832336, -1.0679576396942139, 1.4978454113006592, -1.1095343828201294, 0.19411101937294006, 1.4721457958221436, 0.3743833899497986, 0.9379345774650574, -0.3524335026741028, -0.965961217880249, -0.8209503293037415, -0.8652346730232239, -0.512340247631073, -0.7681664824485779, 1.4752472639083862, -0.4550136923789978, -1.1690092086791992, 0.45139291882514954, 1.7289392948150635, 1.4209825992584229, -0.032607391476631165, 0.15993602573871613, -1.0873013734817505, -1.1793708801269531, 0.30511635541915894, 0.9592872262001038, -0.9629071950912476, -1.3219869136810303, 1.8369951248168945, -3.20349383354187, 0.17077794671058655, 0.17295269668102264, 1.0050326585769653, 0.5780629515647888, -1.0239343643188477, 0.048721615225076675, 1.9987826347351074, 0.5030399560928345, -1.130146861076355, -1.7298696041107178, 1.5713765621185303, 0.7562357783317566, -0.8789254426956177, -0.4725569486618042, -2.0139496326446533, -0.993596076965332, -0.6682624220848083, 0.40927577018737793, -1.5412907600402832, -0.37743690609931946, 1.330116629600525, -0.9824479222297668, 1.3350456953048706, -1.1336798667907715, -0.9542041420936584, -1.6280767917633057, -0.782471239566803, -0.5541797876358032, -0.04595382511615753, -0.09464468061923981, 0.027631647884845734, 0.20503684878349304, 0.4064045548439026, 0.554069459438324, 0.1826825737953186, -0.4136486351490021, 0.07063982635736465, -1.0329021215438843, -1.0626189708709717, -1.4126029014587402, 1.7023524045944214, 1.6688302755355835, 0.8120408654212952, -0.1404494047164917, 0.526012659072876, 0.11176291853189468, -0.38583552837371826, -1.1836628913879395, -1.4830162525177002, 0.52057945728302, -0.08563745021820068, -0.054211895912885666, -1.7076563835144043, -0.37749016284942627, -0.776141881942749, -0.7264940738677979, 0.8720974922180176, 0.8243210911750793, -1.2602732181549072, -1.6500650644302368, 0.3321385085582733, 0.5243268013000488, -0.2152787148952484, 1.4299428462982178, -0.06138024106621742, -0.1719008833169937, -0.47648850083351135, 0.11428527534008026, -1.854809284210205, -2.9406039714813232, 1.1154723167419434, -0.7311429977416992, -1.9311294555664062, 2.614192008972168, 1.5927082300186157, -0.8436919450759888, -0.43502914905548096, 0.6033719182014465, 0.021311810240149498, 0.04499933868646622, 0.6903507113456726, -0.4159016013145447, 0.11462468653917313, -0.22565968334674835, -1.1125679016113281, 0.8070600032806396, -1.2640362977981567, -1.6071140766143799, -2.197826862335205, -1.9894592761993408, -2.0231096744537354, 0.683394730091095, 0.09059440344572067, -0.6275311708450317, 1.2018930912017822, 0.7139859199523926, -2.971508026123047, 0.8889244794845581, 0.12648704648017883, -1.1288394927978516, -1.0685003995895386, -0.939428985118866, -0.28958478569984436, 1.7526156902313232, 0.9958914518356323, 0.472446084022522, 0.6116423010826111, -1.3686411380767822, 0.3612034320831299, 0.7285686135292053, 0.23880890011787415, -3.0611934661865234, 0.01657762937247753, 0.6507482528686523, -0.1390106976032257, -0.4701886475086212, 1.7826182842254639, -1.8196223974227905, -0.2492307871580124, -0.13774189352989197, -0.48256781697273254, 1.3868716955184937, -0.0033420214895159006, 0.12355803698301315, -0.21804095804691315, 0.8671232461929321, -0.00583032937720418, -0.07720198482275009, -0.00020378045155666769, 0.2329261302947998, -0.3431648015975952, 1.5465117692947388, 0.7935453653335571, -0.0032115059439092875, 0.19718386232852936, -3.338965654373169, 0.7365482449531555, -1.12717604637146, 0.29761168360710144, 0.5530610084533691, -0.861932635307312, -0.208005890250206, 0.03320566937327385, -0.6654196977615356, -0.1997584104537964, -1.57794988155365, -0.09811484813690186, 0.12258508801460266, 0.10669245570898056, -0.3441446125507355, 0.04826541244983673, -0.37734150886535645, 0.5041190981864929, -2.6188089847564697, -0.21960142254829407, 0.24227041006088257, 0.4300619661808014, -1.1658174991607666, -1.1322330236434937, 1.3346445560455322, -2.2295970916748047, 0.21555808186531067, 2.1680638790130615, -0.5614678263664246, 0.4793103337287903, -0.9287159442901611, -0.6891378164291382, -1.1836849451065063, 0.18428537249565125, -2.1057019233703613, -0.2812730669975281, -0.20805604755878448, 0.6865155100822449, 0.7489088773727417, 1.5728271007537842, 1.069760799407959, -1.3803720474243164, 1.397406816482544, -0.5402094125747681, 1.1012234687805176, 1.9592626094818115, -1.3970919847488403, 0.93915855884552, 0.08623039722442627, -0.565816342830658, 0.5254994630813599, 0.9153246879577637, -0.34203195571899414, 0.737920880317688, 0.29664671421051025, -1.1533101797103882, -0.8856245875358582, 1.2091518640518188, -0.3638760447502136, -0.1252809762954712, 0.12041568011045456, -1.4351060390472412, 1.486673355102539, -0.9456214904785156, 0.45201194286346436, -2.4699721336364746, -1.3548219203948975, 0.7137414813041687, 0.994803249835968, -0.6238420605659485, 1.6674197912216187, 1.4035162925720215, 0.004036735277622938, -0.6695289611816406, -0.8393425345420837, -1.3195483684539795, 0.07161694765090942, -0.08621594309806824, 1.4022774696350098, 2.0561904907226562, -0.3361293375492096, -0.8401802182197571, 1.1644994020462036, 0.23624084889888763, 0.7748390436172485, 0.8187405467033386, -0.16938364505767822, 0.46315357089042664, -1.1879547834396362, 0.9442821741104126, -0.5380663275718689, -0.29368820786476135, 1.1281355619430542, 1.1571946144104004, -0.7102332711219788, -2.0303449630737305, -0.19232408702373505, -1.0788800716400146, -0.9008455276489258, -1.6498337984085083, 0.603234052658081, 0.6662301421165466, -1.843518853187561, 0.6307636499404907, 1.7166022062301636, 0.8315699696540833, -0.733751118183136, -0.007136473897844553, 1.4893944263458252, 1.6448493003845215, -0.6859825253486633, -0.06303966790437698, -0.1780676245689392, -0.5420191287994385, -0.8676934242248535, 0.5314580202102661, 0.21550968289375305, -0.7484748363494873, 1.350132942199707, -1.028055191040039, 0.84895920753479, 1.1495906114578247, -0.9400877952575684, -0.7670184373855591, 0.6870369911193848, 0.29430127143859863, 0.28620532155036926, -1.7329764366149902, -0.1957150399684906, -0.6016755700111389, 1.014257550239563, -0.02979111857712269, -0.5994584560394287, 0.8024559020996094, 1.2489464282989502, 0.6787270903587341, 0.2820417582988739, -0.8251032829284668, 0.8650197982788086, -0.2930544912815094, -0.4644407629966736, 1.993374228477478, 1.4870121479034424, 0.6485013961791992, 0.8150389790534973, -0.1560312956571579, -1.0107749700546265, 0.8897630572319031, -0.9742038249969482, -0.1250261515378952, -1.3533536195755005, 0.10349877178668976, -1.26442289352417, -0.14798791706562042, -0.40398338437080383, 0.8563410639762878, 0.35355162620544434, 0.052325908094644547, -1.321828842163086, -1.8181991577148438, -0.2832028865814209, -1.1383168697357178, -0.24960295855998993, -0.0014342128997668624, -0.10351572185754776, -0.35586416721343994, 0.2891620695590973, 0.5800673961639404, 0.7201194167137146, 1.3331153392791748, 0.6794443726539612, 0.581721305847168, -0.1165292039513588, 0.36424097418785095, -0.12517096102237701, 1.4845167398452759, 0.7005371451377869, 0.14018507301807404, -0.3714907467365265, -1.194471836090088, 1.3869901895523071, 0.15063397586345673, 0.7974308133125305, 0.9264554381370544, -1.1850486993789673, -0.5981191396713257, -1.211650013923645, -0.2652718722820282, 1.0666931867599487, -0.5677632093429565, 0.4574865698814392, -1.2680269479751587, -0.08205819129943848, 0.6452803611755371, -0.5637145042419434, 0.9945110082626343, -1.12452232837677, 0.641680121421814, -0.34862208366394043, -0.9344270825386047, -0.5118957757949829, 0.13308046758174896, 0.35204359889030457, 0.39741653203964233, -0.48084115982055664, -0.6431153416633606, 0.7533876895904541, -1.0783847570419312, -0.8530959486961365, 0.39490947127342224, -1.5364696979522705, 1.8582309484481812, 0.18092451989650726, -0.1026335135102272, -0.6255730986595154, -0.8324630856513977, -1.7594772577285767, 0.5162707567214966, -0.5550791621208191, 0.7780004739761353, -1.567427158355713, 1.2920455932617188, 0.14221732318401337, 0.36917492747306824, -1.0314981937408447, -0.2737441062927246, 1.1478865146636963, 2.33066463470459, -0.6643918752670288, -0.26464933156967163, -1.017505168914795, -0.04002516344189644, 0.9617705345153809, 0.8260875344276428, 0.398052453994751, 1.2962095737457275, 0.46540936827659607, 1.6322436332702637, -1.7194124460220337, -1.026020884513855, -0.2255202978849411, -2.262005090713501, 0.40248993039131165, 0.03935311734676361, -1.8690452575683594, -0.10196953266859055, 0.3795211613178253, -0.46429601311683655, 1.0375475883483887, 0.15585222840309143, -0.29503345489501953, 1.0430548191070557, 0.5228325724601746, 1.6657702922821045, -1.2020092010498047, 0.7611839175224304, 0.04726066812872887, -1.5236114263534546, -0.8928517699241638, 0.33161461353302, -1.3557184934616089, 0.3840998113155365, 1.1133779287338257, -0.7485037446022034, -0.791433572769165, -0.6039144396781921, 0.08944900333881378, 1.1238609552383423, -1.5219976902008057, -0.9576273560523987, -0.49631184339523315, -0.7930663824081421, -0.7152696251869202, -0.6988310813903809, -0.42570748925209045, -0.6490906476974487, 0.5373982191085815, 2.5418343544006348, -0.3223210275173187, -0.9910289645195007, 0.6411523222923279, 1.1586246490478516, 0.05663459748029709, -1.3150866031646729, -0.723944365978241, -1.2814810276031494, -0.7464870810508728, 1.4668422937393188, -1.5594195127487183, 0.084400475025177, 0.6166091561317444, -0.4127078950405121, -1.3944975137710571, -0.7542908787727356, -0.07852691411972046, 1.7894282341003418, -0.6275452971458435, 1.1770106554031372, -1.2757521867752075, 0.48883479833602905, -0.2478594034910202, -1.7693387269973755, 0.9482967853546143, 0.46295827627182007, 0.20959189534187317, 0.7842419147491455, 0.4079461991786957, -1.1212631464004517, 0.6118330955505371, -1.209585189819336, 1.432543396949768, -1.0125329494476318, -0.6831709742546082, -2.2626686096191406, 1.1565265655517578, -0.23177820444107056, 1.3126057386398315, -1.003989338874817, -0.15723861753940582, -0.4266648590564728, 0.6132369637489319], [1.2126988172531128, 0.4308633506298065, 0.5054478645324707, -0.8359031677246094, -2.2424118518829346, -0.30316781997680664, -0.3409295976161957, 0.847152590751648, 0.5513558387756348, -0.18826167285442352, 0.1039646789431572, 1.1800092458724976, 1.4084274768829346, 1.7081749439239502, -0.9459327459335327, 0.08097624033689499, -0.12125293165445328, -0.44536882638931274, 1.4380230903625488, -0.5172800421714783, 0.3113088309764862, 1.1232430934906006, -0.18345138430595398, 1.683214783668518, 0.0498611219227314, -1.2049769163131714, 0.04437960311770439, -1.1054141521453857, 0.6744562983512878, 1.1067196130752563, 0.5194612145423889, -0.7664662003517151, -0.15548749268054962, -0.3779996335506439, 1.002789855003357, -0.4344806671142578, 1.6115226745605469, 1.299755573272705, 1.3003507852554321, -0.13675859570503235, -0.5276608467102051, -0.45934778451919556, 0.1930939257144928, -2.2268452644348145, -1.7406967878341675, 0.10059384256601334, -0.6189619302749634, 1.2236350774765015, -0.6428460478782654, 0.6370950937271118, 0.5453860759735107, -1.909622311592102, 0.01697695255279541, -1.3749741315841675, 0.22903159260749817, -0.5729140639305115, 0.1724570393562317, 1.052401065826416, 0.42050787806510925, -0.11743749678134918, -1.5227694511413574, 1.804668664932251, 0.7261767983436584, 0.31139451265335083, 1.8525322675704956, -1.510396122932434, 0.629196047782898, -0.34901344776153564, 1.2272106409072876, -1.090706467628479, -0.286612331867218, 0.31594276428222656, 0.3782203495502472, -0.41465964913368225, 0.4723253548145294, 0.8666743040084839, -0.15870651602745056, 0.7379636168479919, 1.1138460636138916, 0.5343665480613708, 0.7100223898887634, -1.150415301322937, -0.7263433337211609, -1.1023457050323486, 0.016934333369135857, 0.7785106301307678, 1.1844637393951416, 0.9661666750907898, -1.2223042249679565, 2.102914571762085, -0.6744180917739868, 0.7985761761665344, 0.28318196535110474, -0.16101346909999847, -0.1838526874780655, -2.3940579891204834, -2.153118371963501, 0.6961767673492432, -0.759705662727356, -0.7183858156204224, -0.9004009962081909, 0.3303948640823364, 1.7321841716766357, 0.4439848065376282, -0.08736620098352432, -0.5792991518974304, 0.37580588459968567, 1.1060479879379272, -1.641241192817688, -0.8338002562522888, -1.4402447938919067, -0.3845064342021942, -0.17428353428840637, 0.7508320808410645, 1.0190825462341309, 1.1604347229003906, 1.0776711702346802, 0.7695714235305786, 0.9097573757171631, -0.38313034176826477, -0.7869116067886353, 1.032922387123108, 0.02464197762310505, 1.8614742755889893, -1.0676417350769043, -0.5600631237030029, 0.797841489315033, 1.6407848596572876, 0.8834973573684692, 0.26128754019737244, -1.5828289985656738, 0.07940147817134857, 0.15524180233478546, 0.26572176814079285, 1.0013717412948608, -0.662871241569519, -0.690801203250885, 0.8200207352638245, -1.4071471691131592, -1.2080674171447754, -0.8996873497962952, -0.0028150840662419796, 0.7415910363197327, -0.6909641027450562, -1.3729896545410156, -0.1081022098660469, 0.21547973155975342, 0.12065203487873077, 0.26563331484794617, -2.2883734703063965, -0.08229278773069382, 0.8504343628883362, -0.9649325609207153, -0.8002945184707642, 2.5311126708984375, -1.701389193534851, 1.115662693977356, -0.08206819742918015, -1.5435580015182495, -0.6600737571716309, -0.18540820479393005, -2.523001194000244, -1.0566964149475098, -0.4509001076221466, 0.47917506098747253, -0.785534143447876, 0.05664430558681488, 0.9909051656723022, -0.5473723411560059, -0.8165642619132996, 1.5263170003890991, 0.5295056700706482, 0.6327090859413147, -0.48148053884506226, -0.018220417201519012, -1.4436548948287964, -0.12338894605636597, -0.8137488961219788, 1.9291117191314697, -0.5951175689697266, 1.0911180973052979, -0.8841977715492249, 0.5744550824165344, -1.1128244400024414, -0.6377103328704834, 0.6221933960914612, 1.2755317687988281, 0.4550977051258087, -1.9729167222976685, -0.2244649976491928, -0.00014840201765764505, 1.4874365329742432, 1.0146173238754272, 0.8674945831298828, -0.8488209843635559, 0.737504780292511, 0.7378964424133301, -1.517162561416626, 0.8302294015884399, 0.06799936294555664, -1.110618233680725, 1.2295451164245605, 0.5843743085861206, -0.5709375739097595, -0.04895477741956711, -0.7032279968261719, -0.13878865540027618, -0.6773021817207336, 0.5713416337966919, -0.5726060271263123, 1.3429945707321167, -2.0525832176208496, -0.6711971163749695, 0.8864949941635132, -0.1543993502855301, -0.08531433343887329, 0.20173412561416626, -0.06844796240329742, 1.6214255094528198, 0.47854864597320557, 0.5149669647216797, -1.1171321868896484, 0.3320602476596832, 1.0204353332519531, -1.2011090517044067, 0.8312864899635315, -0.4541175961494446, -0.4342367649078369, -1.5733879804611206, 0.048055581748485565, -1.0916037559509277, -0.5084810853004456, -0.5570777654647827, -0.05958785116672516, 0.0622297041118145, 0.9961419105529785, 0.5102606415748596, 0.8320412635803223, 0.263008177280426, -0.467509925365448, 1.0224974155426025, -1.345997929573059, -0.10901650041341782, -0.010873277671635151, -0.4773688316345215, -1.3912575244903564, 0.6182817220687866, 0.26992693543434143, -0.8786266446113586, -0.27865707874298096, 0.46463075280189514, 0.32720792293548584, 0.3499116897583008, -0.8733217716217041, 1.9867656230926514, -1.3743383884429932, 0.20567713677883148, -0.93837571144104, -2.3043594360351562, 0.4356171786785126, -0.11503395438194275, 0.575832188129425, 0.6328285932540894, -0.03775754198431969, -0.7260038256645203, -2.116689682006836, 0.04661419242620468, -0.04896455258131027, -0.9188101887702942, -0.83514004945755, -1.3517334461212158, -1.1460142135620117, -0.5199181437492371, -0.351796954870224, -0.2658214569091797, 1.4486913681030273, -0.15035958588123322, -0.4686525762081146, 0.6902673840522766, -0.7698236703872681, -0.38970157504081726, -0.42960649728775024, 1.0864306688308716, 0.6983479261398315, -0.7521203756332397, 1.0542786121368408, 1.4350649118423462, -0.5475247502326965, -0.6191763281822205, 0.5491483211517334, -0.9468052983283997, 1.0488234758377075, -0.8519839644432068, -1.2224090099334717, 0.8470685482025146, 1.5908085107803345, 0.48015135526657104, -1.514035701751709, -0.7186359167098999, 0.9088801145553589, -0.2770756483078003, -1.5088534355163574, -0.39492326974868774, -0.6408520340919495, -0.9656771421432495, -0.4598214626312256, 1.8417699337005615, -0.01436553243547678, -1.556607723236084, 0.4237508475780487, -0.9165841341018677, -0.19608454406261444, -1.4672013521194458, -2.2504796981811523, 0.8853110074996948, 0.22484596073627472, 0.20599249005317688, 0.6473838090896606, 2.0738275051116943, -0.40704330801963806, 0.30854684114456177, -1.2513196468353271, 1.42677640914917, -0.36852920055389404, -0.9070609211921692, 1.9519699811935425, -1.7567747831344604, 1.8161832094192505, -0.09516259282827377, 0.1884237825870514, 0.0028474077116698027, 0.6685178875923157, 0.7030537128448486, -0.6821029186248779, -0.06052505597472191, 0.7445729970932007, -0.3314339220523834, 1.0687609910964966, 0.8147184252738953, 0.3478236794471741, -0.49797770380973816, 2.1061902046203613, -0.7930893898010254, -1.280540943145752, -1.737045407295227, 0.5837386250495911, 0.9719851016998291, 0.001445653149858117, 0.658176839351654, -0.3119995594024658, -0.6186580657958984, 1.4144645929336548, -0.06046116352081299, -0.6338104605674744, 1.9402847290039062, -0.20864666998386383, -0.598767876625061, 1.1581286191940308, -0.13412420451641083, -0.06142404302954674, 0.27003714442253113, -2.1935555934906006, 0.5922665596008301, 0.6184872984886169, 1.7877087593078613, 1.389962911605835, 1.310613751411438, 1.9905937910079956, -0.5884559154510498, 1.9688084125518799, -0.36075901985168457, -1.0769060850143433, 0.632800281047821, -0.1878574639558792, -0.5935847759246826, 0.6951115727424622, 1.3676031827926636, 1.7868216037750244, -0.3074120581150055, 0.5249260663986206, 0.2987375557422638, -0.26062241196632385, 0.36707964539527893, 1.1712493896484375, 0.8797813653945923, 2.0676610469818115, -0.7114371657371521, -1.6122286319732666, -1.5709089040756226, -2.0727617740631104, -0.8292421698570251, 0.13884803652763367, -0.19304129481315613, -0.43394553661346436, -1.1049492359161377, -0.7831997275352478, -0.2675842344760895, 0.2777805030345917, 0.7387800812721252, 1.4320579767227173, -0.05517717823386192, -1.327115535736084, -1.579206109046936, 1.2699668407440186, 1.3754494190216064, -0.5773056149482727, -1.6370903253555298, -0.12124457955360413, -0.5009398460388184, 0.54746413230896, 0.26365551352500916, -0.49554088711738586, 1.274505615234375, -0.960092306137085, -2.5533199310302734, -1.0539048910140991, 0.29341188073158264, -0.5636165738105774, -3.1040871143341064, 0.27564311027526855, -0.48557156324386597, -0.5117404460906982, -0.36476659774780273, 0.6413136720657349, -0.48260989785194397, -1.9100345373153687, 0.39693742990493774, 0.659534752368927, -1.047055721282959, 0.8709642291069031, -1.5325963497161865, -1.222299575805664, -1.174704909324646, 0.07932513952255249, 0.10183224081993103, -0.44003576040267944, -0.5541620254516602, 1.120713710784912, 2.4777259826660156, -1.4668879508972168, -1.4272871017456055, -0.7932406663894653, 1.6152986288070679, -1.0439485311508179, -1.2265336513519287, 0.14342576265335083, 0.4642975628376007, -1.700325846672058, 0.7088112831115723, -0.11323390156030655, 1.1402697563171387, -0.6237022876739502, 0.7409945130348206, 0.6700093746185303, 0.856239914894104, 0.6624008417129517, -0.7645136117935181, -1.6157152652740479, -1.6655932664871216, -0.5266454815864563, 2.6946518421173096, -1.262938380241394, 2.242441415786743, -0.24096930027008057, -1.381319284439087, 0.5160351395606995, -0.9220035672187805, -0.2602537274360657, -0.038723647594451904, 0.7097393870353699, 1.1274906396865845, -0.1509782373905182, 0.5168001651763916, 0.04001521319150925, -0.2279224395751953, 1.1078526973724365, -0.8038370609283447, 0.007037760224193335, -0.5256094932556152, -0.5855196714401245, 0.45927420258522034, 0.5773605704307556, -0.04170874133706093, -0.16941198706626892, -0.9144890308380127, 0.08748555183410645, -0.3831745386123657, 1.439849615097046, -0.846880316734314, 1.262072205543518, 0.6503556370735168, 1.4970414638519287, 0.985736608505249, -0.39453452825546265, -0.6717882752418518, -0.35081011056900024, 0.737546980381012, -0.9043987989425659, -0.13981974124908447, 0.8720025420188904, -0.12394282966852188, -0.12238826602697372, -0.20453022420406342, 0.9089158773422241, 0.22180375456809998, -0.5975184440612793, -0.26849183440208435, -1.1878492832183838, -0.09052897244691849, 0.06853868812322617, -0.0650336891412735, 0.9646174311637878, 0.8666733503341675, 0.06039034575223923, 0.4354444444179535, 0.9841194152832031, -1.045621633529663, -0.8742444515228271, 0.06259310990571976, 2.7896296977996826, -0.4088028073310852, 0.017639514058828354, -0.9857781529426575, -0.5844792723655701, 0.3392372131347656, -2.450025796890259, 0.42778798937797546, -0.5202094912528992, -2.0840699672698975, -0.5693126916885376, -1.0335575342178345, 0.5968713164329529, 0.5550835132598877, -0.8756784200668335, -1.4907257556915283, 0.1974913775920868, 0.3662709891796112, 0.2936100661754608, 0.7409557700157166, -0.6392293572425842, 0.7842687368392944, 0.5011973977088928, 0.6096161603927612, -1.9656929969787598, 1.2514220476150513, 0.09524404257535934, -0.9690325856208801, 0.17116199433803558, 0.4175046682357788, -1.878954529762268, -0.2233792096376419, 1.972490668296814, -0.42472201585769653, 0.6022768020629883, -0.06467603892087936, 0.3168768882751465, 1.7670847177505493, -0.347765177488327, -0.36667394638061523, 0.4498564302921295, -1.3717185258865356, -0.2846793234348297, 0.27773430943489075, -1.2565009593963623, -0.12015853822231293, 1.6303210258483887, -0.09081728011369705, 0.1322987973690033, 0.26139435172080994, -0.3528979420661926, 0.322195440530777, 0.14343132078647614, -0.34420013427734375, -1.1641846895217896, -0.8019683361053467, -1.8521363735198975, 0.1559850126504898, 0.540621817111969, 0.42130017280578613, 0.7854661345481873, -0.9150189757347107, 0.7152324914932251, 2.4155147075653076, 1.041384220123291, -0.4369508922100067, -1.1248236894607544, 0.1965084820985794, 0.018823083490133286, 0.08058922737836838, -1.2762088775634766, -2.4602103233337402, 1.9144980907440186, 0.8431701064109802, 0.32493308186531067, -0.2787836194038391, -0.7323709726333618, -1.682655692100525, -1.5126070976257324, 1.087216854095459, 1.2673728466033936, -1.2529174089431763, 0.4778791666030884, 0.746364951133728, 0.8812850117683411, 0.029655279591679573, -0.3497679531574249, -0.6349067687988281, -0.7818489074707031, 0.8595119118690491, 0.6786710619926453, 0.5056606531143188, -2.0769357681274414, 0.1529758721590042, -0.2085801213979721, 1.3290166854858398, 1.4117472171783447, 1.0960620641708374, 0.05645481124520302, 1.245118260383606, -0.8678721785545349, -0.7426560521125793, -0.5097295641899109, 1.8216302394866943, 0.7700386047363281, 0.829372227191925, -3.289170503616333, -1.3654375076293945, 1.252819299697876, -0.04989162087440491, -1.3469312191009521, 1.7225028276443481, 0.97965008020401, 0.41660773754119873, -1.2213876247406006, -0.3634347915649414, 0.3573058843612671, 0.5513478517532349, 0.23129941523075104, -0.4878009855747223, -1.4837151765823364, -1.3157609701156616, 0.3086954951286316, -0.2747235894203186, -1.3001296520233154, -0.4502105712890625, -0.3194032609462738, 0.253767728805542, -0.4765471816062927, -1.4190205335617065, -1.1972602605819702, 0.8008410334587097, 1.0142933130264282, -1.0499842166900635, 0.6071539521217346, -0.21140357851982117, 0.202947199344635, -2.871788501739502, -0.4614286720752716, 0.05197213590145111, 0.5913600325584412, -0.3113801181316376, 1.2066941261291504, -1.4431174993515015, -0.5368450284004211, 1.2101454734802246, -1.524997591972351, -1.4927558898925781, 0.2819851338863373, 0.6655692458152771, -0.08515545725822449, 0.49251148104667664, 0.6288842558860779, -0.18794307112693787, 0.649322509765625, -1.126030683517456, 0.16309264302253723, 0.09598556160926819, 1.0365031957626343, -0.2803463935852051, -0.6543315649032593, -0.9495293498039246, -0.005280230659991503, 1.0000954866409302, -0.21565568447113037, 1.188637614250183, 1.2484592199325562, -0.22432996332645416, 0.7142279148101807, -0.38545697927474976, -2.308657646179199, 1.4800158739089966, -0.2789003849029541, -0.36985766887664795, 1.2288482189178467, -0.7173395156860352, 0.40648478269577026, 0.6399312615394592, 0.6496551632881165, -1.2351043224334717, -1.8077871799468994, -0.32285380363464355, 0.9101213216781616, 0.28055745363235474, -0.47328221797943115, -0.25668811798095703, -0.5663540363311768, 2.1812806129455566, 0.600525438785553, -0.7512651085853577, -0.100473552942276, 0.5664769411087036, -0.32344940304756165, -0.3266313076019287, 1.8555827140808105, 0.51282799243927, -0.08610061556100845, 1.2269741296768188, -1.1639786958694458, 0.4452647864818573, 0.03934726119041443, 1.0595920085906982, 1.7990517616271973, -0.1628589630126953, 0.5335468649864197, 0.22675231099128723, -0.6504421830177307, -0.210605189204216, -1.0223866701126099, -0.5157335996627808, 1.1006348133087158, 0.7551097869873047, 0.7926074862480164, -0.6677396297454834, 0.9923784136772156, -1.1818410158157349, 0.3542388677597046, -0.9083520174026489, -1.3815343379974365, 0.5860455632209778, -1.0392206907272339, -0.6005692481994629, 0.10880135744810104, 1.486495018005371, 0.5686657428741455, 1.1195404529571533, 0.3400310277938843, -0.38365206122398376, 1.7402448654174805, 0.8282731175422668, 0.46966537833213806, -0.07991036772727966, -1.2462421655654907, -0.5204774737358093, 0.7224290370941162, -0.23625005781650543, -0.9933586716651917, 0.30521440505981445, 0.12431719154119492, -0.6860243678092957, 1.6693428754806519, -2.0877106189727783, 0.5467994213104248, 0.030314447358250618, 1.5206093788146973, -0.28821638226509094, -0.23878213763237, -0.5280942916870117, -0.9026330709457397, 1.2815972566604614, -0.5637855529785156, 0.5758408904075623, -0.6351435780525208, 0.8194578886032104, 0.42338827252388, 2.4242446422576904, -0.7373188138008118, 0.4032527506351471, 0.8492305874824524, -0.06277542561292648, 0.6114708185195923, -1.9407709836959839, 0.3165114223957062, 1.172677993774414, 1.6406078338623047, 0.8169886469841003, 0.7157182693481445, 1.4072214365005493, 0.7039206624031067, 0.23906323313713074, -1.2748767137527466, -0.35712116956710815, 0.7335550785064697, 0.017171477898955345, -1.314584493637085, -1.155233383178711, -1.6514716148376465, 0.026043428108096123, 1.540143609046936, 1.2997111082077026, -0.5336675047874451, 1.5851527452468872, 0.177860289812088, -1.101378321647644, 0.04417137801647186, 1.444818139076233, -1.0291701555252075, -0.4077320992946625, 0.21300376951694489, 0.32330814003944397, -0.5728726387023926, 0.3890697658061981, 0.07525039464235306, -1.968859076499939, -2.0276384353637695, 0.40979474782943726, 0.5256572961807251, -0.1896173357963562, -0.35163411498069763, 0.48151668906211853, -0.6056639552116394, 0.4478021264076233, -0.21833056211471558, 0.6128395795822144, -0.36099255084991455, -0.42809000611305237, -0.2523842751979828, -0.8571651577949524, 1.4049861431121826, 1.6401082277297974, -1.3662742376327515, 0.27946344017982483, 0.16314829885959625, 0.138128399848938, 0.18232744932174683, -1.6327158212661743, 0.5127462148666382, 1.028437852859497, 0.22761766612529755, -1.0469495058059692, -1.0102014541625977, 0.5284908413887024, 1.7580974102020264, -0.481711208820343, -0.16254276037216187, -1.715521216392517, -0.26546111702919006, -1.1771888732910156, 1.1529934406280518, 1.0520236492156982, -0.6509761810302734, 1.186241865158081, -0.032190192490816116, -0.7518448829650879, -0.8254057168960571, 1.0668760538101196, 0.11463961750268936, 0.834010660648346, -1.3743035793304443, -1.1795371770858765, -0.0496707484126091, 0.2691017985343933, -0.7264501452445984, -1.0291779041290283, 0.77619868516922, 0.18919681012630463, -0.6856017708778381, 0.6141571998596191, -1.6344298124313354, -0.6340761780738831, -0.02201758325099945, -0.9687380194664001, -0.6653187870979309, -0.6266435384750366, -0.9920950531959534, 0.22899718582630157, 0.47457778453826904, -0.15748347342014313, -0.47170260548591614, 0.7754368782043457, 0.9554062485694885, -0.14682051539421082, -0.7077320218086243, 0.07075205445289612, 0.42349573969841003, 0.08957496285438538, 0.7866169810295105, 0.21232973039150238, -0.31604182720184326, 1.1920148134231567, -0.30227696895599365, -0.7684809565544128, 1.5525356531143188, 1.8385361433029175, -0.761167585849762, -0.2844167649745941, -0.38171058893203735, -0.008360947482287884, -0.9927849173545837, 0.8985096216201782, -0.5129806399345398, 0.05353708937764168, -1.742438793182373, 0.076835498213768, -0.9883895516395569, -1.0657213926315308, -0.258578360080719, -0.0751333236694336, -0.36264845728874207, 2.0260651111602783, 1.0477148294448853, 0.14866282045841217, 0.47173061966896057, -0.5082455277442932, -0.6475273966789246, -0.9002902507781982, -1.3428817987442017, 0.12414941936731339, 0.3627578020095825, 2.4474971294403076, 0.8110222220420837, 0.2984158992767334, 1.8133869171142578, -0.5509039759635925, -0.642562210559845, -0.7682937979698181, 0.7170644402503967, -0.7029690742492676, 0.19172045588493347, -0.34625503420829773, 0.38604989647865295, -0.8203738927841187, 0.5622408986091614, 1.8311635255813599, 0.7283098697662354, -0.23394139111042023, 1.656352162361145, 2.123697280883789, 0.01889854110777378, 0.21972638368606567, 2.5539586544036865, -0.7897699475288391, -0.14046427607536316, -0.10154825448989868, 0.3710184097290039, 0.8080893754959106, -1.2392494678497314, 0.2539662718772888, -1.4777518510818481, 0.8998855352401733, 1.0331090688705444, -1.7656748294830322, 1.636435866355896, -0.7139608263969421, -0.5002321004867554, 1.4261548519134521, 0.18547622859477997, -1.8844062089920044, 0.23004058003425598, -1.8801580667495728, 0.34876954555511475, 0.5312221646308899, -1.9243971109390259, -0.34379178285598755, -1.0174652338027954, 0.9810112714767456, 1.9310439825057983, -0.46365588903427124, -0.3409024775028229, 1.079849123954773, -0.3037909269332886, 0.8467513918876648, -1.400083065032959, 0.2807517647743225, -0.9073812961578369, -0.6377604603767395, -0.8728071451187134, 0.5915382504463196, -0.5226752758026123, 0.464021772146225, 0.422087162733078, -0.8242278695106506, 1.0769457817077637, -0.06313282251358032, 0.9505615830421448, -0.18374024331569672, 0.6767157316207886, -0.6949664950370789, -0.9789960384368896, 1.0128902196884155, -0.8145075440406799, 0.28331807255744934, -0.2765830457210541, -2.2761852741241455, 1.028570294380188, 0.01889677532017231, -0.4661944508552551, -1.150378942489624, -0.5781947374343872, 0.24762339890003204, -1.719684362411499], [0.08096856623888016, 0.7040568590164185, -0.26601800322532654, -0.15528379380702972, 0.08997955173254013, -1.1939359903335571, 1.7910690307617188, -0.42974498867988586, -0.10904143005609512, 0.48279869556427, 0.050122518092393875, -0.9382461905479431, -2.0459961891174316, -0.929764986038208, 0.6628103852272034, -0.2997172176837921, 0.1890343874692917, -1.173271894454956, -0.5225411057472229, 0.7450114488601685, 0.41640999913215637, -0.017319608479738235, 0.5545940399169922, 1.0938482284545898, 0.7829475402832031, -1.4898185729980469, -1.6304916143417358, 0.0634719729423523, 0.017245052382349968, 1.3178070783615112, -1.3091020584106445, -0.8578673005104065, -1.4635103940963745, -0.8669310212135315, -0.4071793556213379, 0.7395777106285095, 0.7396093010902405, 0.8897613286972046, -0.6803699731826782, 0.32315418124198914, 2.3041985034942627, -0.43516743183135986, 1.8708308935165405, 0.006565852556377649, -1.932437539100647, 0.7194249033927917, -1.290490746498108, -1.9449756145477295, -0.21339556574821472, 2.5177676677703857, -0.9037639498710632, -0.7875957489013672, -1.991869568824768, -0.45454004406929016, -0.11230500787496567, -0.41261494159698486, -1.2718647718429565, -1.1724183559417725, 0.1282176822423935, 0.018863318488001823, 0.2929995059967041, -0.662880539894104, 0.028609085828065872, 0.7134577631950378, -0.7175578474998474, -2.835148811340332, 0.11890514940023422, 1.3763128519058228, 0.14390981197357178, 2.603348970413208, 1.4110952615737915, -0.062486305832862854, 0.6144025325775146, -0.041476648300886154, -1.1418907642364502, 0.6467356085777283, -0.011887075379490852, -0.04541536793112755, -1.0574392080307007, 0.059362318366765976, 0.5056253671646118, -1.114938497543335, -0.08115828782320023, 0.7017647624015808, 1.1288599967956543, 0.19923177361488342, -1.302202820777893, 0.4964459240436554, 1.2044434547424316, -2.0560379028320312, 0.8672917485237122, -0.09644881635904312, -0.6196387410163879, -0.5610296726226807, -2.0710928440093994, 1.4673024415969849, 0.872819185256958, -0.08644477277994156, -0.3649426996707916, 2.1696345806121826, 0.19399937987327576, -0.8779847621917725, -0.5491775274276733, -0.9944746494293213, -0.2580639719963074, -1.3182179927825928, -1.4546483755111694, -0.6106313467025757, 0.522925853729248, -0.01834731735289097, -0.29948943853378296, 0.7294337153434753, 0.06678681075572968, 1.6077784299850464, -0.10030889511108398, -0.4025828540325165, 0.9046205878257751, 0.5975444316864014, -0.27763044834136963, -0.38518160581588745, -0.6265319585800171, 0.4817288815975189, 0.28502050042152405, -1.276387095451355, -0.1946234256029129, -2.209292411804199, 0.8044484853744507, -0.22192083299160004, -2.29660964012146, 0.9517580270767212, -0.21689775586128235, -0.9638002514839172, 0.8160037994384766, -1.6122463941574097, 0.0065564061515033245, 0.7905176877975464, 1.5488340854644775, -1.232264757156372, -0.7542755007743835, 0.284088671207428, -0.4246036410331726, -1.1445555686950684, 0.6767407655715942, 0.5569353699684143, -0.13609853386878967, -1.50267493724823, -0.23351776599884033, -0.22283171117305756, 0.5166322588920593, 0.175307035446167, -0.5102925896644592, -1.179060935974121, 0.3361251950263977, 0.45679113268852234, -2.2238996028900146, 0.16545045375823975, -1.4892367124557495, 0.4855239987373352, 1.1919193267822266, 1.9333055019378662, -0.6013442873954773, 2.534607410430908, -1.0327695608139038, -0.9850857257843018, -1.0506634712219238, -0.27634304761886597, -0.02564479224383831, -0.48853856325149536, -0.03709156811237335, 1.6168333292007446, 1.0823026895523071, 0.3735462427139282, 0.6043100953102112, 0.6198729872703552, -0.012215723283588886, 1.7534977197647095, -0.005281231831759214, 1.2898133993148804, 0.9992744326591492, -1.3385370969772339, -0.2189401239156723, -0.6393561363220215, 0.027000874280929565, 1.3562583923339844, 1.3210350275039673, -0.24757373332977295, 1.4650520086288452, 1.4336268901824951, -0.3264545202255249, -1.8534283638000488, -1.393500804901123, -2.118173599243164, 0.7639492154121399, -0.5664054155349731, -1.1555919647216797, -0.8623709082603455, 0.23335856199264526, 0.0005872324691154063, 0.056331291794776917, -1.684543490409851, -0.3839738070964813, -0.33809712529182434, -1.9571055173873901, -0.3548654615879059, 1.8348413705825806, 0.0857132151722908, 0.5470324754714966, 0.8647376298904419, 1.2442271709442139, -1.3174057006835938, -0.4681393802165985, -2.0176522731781006, 0.16520294547080994, -0.012418034486472607, -1.654663324356079, 0.16674360632896423, -0.7833678722381592, -0.4290275275707245, -0.35363879799842834, -0.6451564431190491, -1.1519063711166382, -0.8362201452255249, -0.6914778351783752, -0.6189651489257812, -0.7156113386154175, -0.7369735240936279, 1.182378888130188, -0.5578328967094421, 0.23544564843177795, -0.46836352348327637, 1.0472670793533325, 0.1412377953529358, 0.2068081498146057, 2.9524316787719727, 0.31226444244384766, -1.6522761583328247, 0.21245485544204712, -0.249663844704628, 0.4471849799156189, 1.254180669784546, 0.7531376481056213, -0.13648590445518494, 1.7943804264068604, -1.016892433166504, 0.7442684769630432, 1.0164600610733032, -0.46998330950737, -0.9184712767601013, 1.0990517139434814, -1.9392619132995605, 1.1837204694747925, 1.9937664270401, 1.9712382555007935, 2.584498167037964, 1.9835771322250366, -0.3295137286186218, -1.5990970134735107, -2.526376962661743, 1.1510884761810303, -0.016992181539535522, 0.11002139747142792, 0.7380950450897217, -1.3981496095657349, 1.3787972927093506, 0.507006824016571, -0.3671753704547882, -1.2340930700302124, -0.7966349720954895, -0.08990868926048279, 0.4477900266647339, 0.940574586391449, 1.060347318649292, -2.279217004776001, 0.359675794839859, -0.591613233089447, -0.309993714094162, 1.012683391571045, -1.5186877250671387, -0.1429443657398224, -0.7889883518218994, 0.8770788311958313, -1.6503889560699463, 0.2023889422416687, -0.12543530762195587, 0.0863252505660057, 0.8318970799446106, 0.5288181900978088, -0.016470618546009064, -0.6810449361801147, 1.0903092622756958, 0.22343000769615173, 1.5624139308929443, 0.1031491607427597, 2.1349682807922363, 0.5042309761047363, -0.8958699703216553, 1.4873205423355103, -0.12374896556138992, 0.3100937306880951, -2.009127140045166, 1.11051607131958, -0.9030401110649109, -0.2726895213127136, -1.540191650390625, 1.7284969091415405, 1.168027400970459, -0.23162321746349335, 0.4999687671661377, -0.257422536611557, -0.8832851052284241, -1.577312707901001, -0.29193174839019775, -0.5488315224647522, 1.4662047624588013, 0.44267141819000244, 1.4232470989227295, -0.6699269413948059, -0.9386476874351501, 0.2370959222316742, -1.0235878229141235, 1.7789455652236938, 0.6599888801574707, -0.039682772010564804, 1.0824124813079834, 0.48264309763908386, -0.28193292021751404, 0.9381414651870728, -0.15817691385746002, 0.6454399824142456, 0.2777183949947357, 0.12309571355581284, -0.8376158475875854, 0.0513443797826767, 0.9160772562026978, -0.6061962246894836, -0.7238214612007141, -0.21575650572776794, -0.03881838917732239, 0.10454650968313217, 0.2608238458633423, -1.0042401552200317, 0.33330661058425903, 1.4215748310089111, 0.2207064926624298, 0.5412224531173706, -0.15387262403964996, 0.9166146516799927, 0.7077814936637878, -0.5163993239402771, -0.521397590637207, 1.3077397346496582, -0.44575411081314087, 0.8891177177429199, 1.4789930582046509, 1.089962124824524, -0.38057437539100647, -2.1005427837371826, 1.2629565000534058, -0.8801807761192322, 1.9687916040420532, -1.7622531652450562, 0.0008322698995471001, -0.7036281228065491, 0.334137499332428, 0.15012969076633453, -0.4821166396141052, -0.40192338824272156, -1.3733924627304077, 0.6098632216453552, -0.7134799361228943, 0.7606937289237976, -0.1678341180086136, 0.03247465565800667, 0.28782960772514343, 1.5667020082473755, 1.1060391664505005, 0.07000382244586945, -1.1424909830093384, -0.6288101673126221, 0.06649741530418396, -0.3832452893257141, 1.3905361890792847, 0.6148890256881714, 1.8775238990783691, 0.6532477736473083, -0.3434807360172272, 0.2884668707847595, -2.9918813705444336, -0.03838120028376579, 0.0895669162273407, 0.2163393348455429, 0.4821345806121826, 0.36927565932273865, 0.20820966362953186, 1.6241679191589355, 0.24265846610069275, -0.6549892425537109, -1.227137804031372, -0.702027440071106, 1.2281415462493896, -1.532724380493164, 1.6730451583862305, 0.16524340212345123, -2.494471311569214, 2.182762622833252, -1.111668586730957, -0.02125295251607895, -0.08519970625638962, 0.3640328347682953, -0.7925217151641846, 0.07662498205900192, -1.0325648784637451, 1.0761581659317017, 0.5264270901679993, -0.49714136123657227, 0.06498579680919647, -0.39349931478500366, -0.7436093688011169, 1.821386694908142, 0.5901083946228027, 1.2119390964508057, -0.5981799364089966, -2.2160136699676514, -0.5771565437316895, 0.2809673249721527, -1.034464716911316, 0.2420966774225235, 0.1350654512643814, 1.6168665885925293, 0.9655808210372925, -0.895240068435669, -0.6681346893310547, 0.23737022280693054, 0.25679832696914673, -0.7589268684387207, 0.3914366066455841, 0.16396868228912354, 0.01752590201795101, 2.0831873416900635, -0.3168998658657074, -1.6583197116851807, -0.06626948714256287, 0.4969882369041443, -0.6996809244155884, -0.5290704965591431, 1.6677875518798828, 1.4178099632263184, 0.2671836316585541, -0.891757071018219, -0.31771618127822876, -0.45372188091278076, -0.041216395795345306, -0.14010010659694672, 1.3682299852371216, -0.6890897154808044, -0.5339503884315491, 0.8422579169273376, 0.28155311942100525, 0.04878270998597145, 0.679031491279602, -0.18141509592533112, 1.0080170631408691, 0.4217769205570221, -0.038650840520858765, 0.5140620470046997, 0.4204712510108948, 0.40364542603492737, -0.4826849400997162, 0.4553822875022888, 0.1533948928117752, -0.15211749076843262, 0.613062858581543, -0.6639817953109741, -1.4974254369735718, 0.3759380877017975, 0.2715650796890259, 1.2818191051483154, -0.5482725501060486, -0.2789331376552582, 0.6957245469093323, 0.3223193883895874, -0.8194023966789246, -1.0681769847869873, -0.5565917491912842, -0.4257323443889618, -0.9346981644630432, 0.42051586508750916, 0.5782440304756165, 0.42728471755981445, 0.6326881647109985, 2.8491992950439453, 1.6995307207107544, 1.0953271389007568, 1.6153894662857056, 0.07671705633401871, -0.4427776038646698, 1.0648337602615356, -0.8593285083770752, 0.2780531644821167, -0.5854749083518982, -1.0440937280654907, -1.8285173177719116, -0.48805949091911316, 0.9442237615585327, -1.0386285781860352, -0.5664021968841553, 0.4582250714302063, 1.6124248504638672, 1.1361517906188965, -1.1222355365753174, -1.370438814163208, 0.9467796683311462, 0.7444551587104797, -0.554362952709198, 0.9404987692832947, -1.7871371507644653, -0.1474931240081787, 0.02467566542327404, 0.7628772258758545, 0.09128660708665848, -1.3052698373794556, 0.04214268922805786, -0.008811688981950283, -0.974420428276062, 0.6381734609603882, -0.27245643734931946, -0.11262784153223038, -0.45346787571907043, 0.028694549575448036, 0.8180105686187744, -0.7516298890113831, 1.6868342161178589, 1.8291422128677368, 0.5692669153213501, -0.5848615765571594, -0.42422670125961304, 1.2460819482803345, -0.2530021071434021, 0.3023177981376648, -1.5817360877990723, -0.43807199597358704, 0.7389644980430603, -0.5225156545639038, -0.15799777209758759, -1.6200770139694214, 0.7408314943313599, 0.8872646689414978, 0.247273251414299, 0.818847119808197, -0.7511676549911499, -0.0987505167722702, 0.8332277536392212, -0.11961206793785095, -0.2633900046348572, -1.1268868446350098, -0.3795892596244812, 0.5405398607254028, -0.5068516731262207, 1.2607020139694214, -1.131170630455017, 0.9418520927429199, -1.5835148096084595, -1.8175357580184937, -1.7986414432525635, 1.062795639038086, -0.3735370635986328, 2.0761187076568604, 0.09958021342754364, 0.40285149216651917, -0.486655056476593, -2.068366050720215, -1.3703913688659668, -0.9628908038139343, 0.311718612909317, 0.43205782771110535, -0.7653559446334839, 0.40927475690841675, -1.6187973022460938, 1.4480737447738647, -0.45053696632385254, -1.0462144613265991, 0.537277102470398, -0.042656801640987396, -0.6163292527198792, 0.6935628652572632, -0.16492995619773865, -1.7383280992507935, -0.2181292176246643, -0.3693705201148987, -0.3465580344200134, 0.15447209775447845, -1.7155125141143799, 1.2578126192092896, -0.5755497217178345, -0.6338433027267456, -0.6855018138885498, -0.45403027534484863, -1.083532452583313, -1.3660552501678467, -0.0524374283850193, 2.291614532470703, -0.7858783602714539, 0.5403785109519958, 2.4409193992614746, 0.9347535967826843, 0.7235860824584961, -0.18747876584529877, -1.248671054840088, -0.53156977891922, 1.1778429746627808, 0.414406418800354, -2.1056160926818848, 0.5314387083053589, 1.693809151649475, -0.25330793857574463, 0.4704681634902954, 0.8912156820297241, 1.132522702217102, -1.0392074584960938, -0.11981943249702454, 0.8676980137825012, -0.06845656037330627, -0.26142722368240356, 0.6397309899330139, -0.2631099820137024, 1.5986794233322144, 0.087487131357193, 0.9678609371185303, -0.19892749190330505, -0.7433308959007263, -1.3311595916748047, 0.25076648592948914, 0.9845492243766785, 0.22506199777126312, 1.3628630638122559, 1.4082540273666382, -1.5018221139907837, 0.823339581489563, 0.0037668277509510517, 0.6667127013206482, 0.34382903575897217, -0.4823755621910095, -1.2211705446243286, 0.37127459049224854, -0.8538988828659058, -2.293006420135498, 0.01960199698805809, 1.5359395742416382, -1.3909294605255127, 0.11298931390047073, -0.4971473515033722, -0.3592670261859894, -0.7672198414802551, 0.983087420463562, 1.6401950120925903, 0.48264607787132263, 0.46961039304733276, -0.41519638895988464, -1.2753376960754395, -0.433283269405365, 0.011841265484690666, 1.0417181253433228, 1.0948450565338135, -0.36913779377937317, -0.6545405983924866, 1.3360319137573242, -1.4020497798919678, 1.094440221786499, 0.7473188042640686, 1.115639090538025, 0.5630382299423218, 2.0377650260925293, -0.9462432265281677, 0.2473585605621338, 0.967562198638916, -1.6617592573165894, -1.1191025972366333, -0.10509845614433289, -0.5746841430664062, 0.4210239052772522, -0.2123369574546814, 0.05981852859258652, 0.39104777574539185, 0.5055533051490784, -0.6235424876213074, 0.6942844390869141, 0.22732718288898468, -0.06426264345645905, -0.024505507200956345, 0.31080418825149536, -1.2962937355041504, -1.5994991064071655, 2.17903733253479, 0.7147895693778992, 1.0484200716018677, 1.8453577756881714, -0.5277562141418457, -1.250253438949585, -0.8098965883255005, 0.31209415197372437, 1.36568284034729, 0.24939189851284027, 1.1011556386947632, 0.7521063685417175, -0.19548168778419495, 0.4821014106273651, 0.1836971789598465, 0.7488990426063538, 0.1320403814315796, 0.24584607779979706, 0.9505666494369507, -0.3991522192955017, 1.8610482215881348, -0.06075328588485718, 0.6161274313926697, -1.705392599105835, -0.2974565923213959, 0.08116091042757034, 0.052488286048173904, -1.4325013160705566, -0.18336665630340576, 0.2028639167547226, 1.2990353107452393, 1.371488094329834, 1.0402960777282715, -0.01703568734228611, 0.7675744295120239, -0.539462149143219, -1.0132575035095215, -0.5803447365760803, -0.143559992313385, 0.599380612373352, 0.8277239799499512, -0.029162032529711723, 0.5870388746261597, -0.4357321858406067, -0.09173116832971573, 0.41522759199142456, 0.48336365818977356, 0.1573290079832077, 1.1617014408111572, 0.6617444753646851, -0.9270385503768921, -0.6850554943084717, -1.1003459692001343, -0.1848912090063095, 0.637992262840271, -1.5474281311035156, 0.3423903286457062, -0.6758543848991394, -0.5262289643287659, 0.19560816884040833, -0.1328137069940567, 0.30117323994636536, -1.2727792263031006, -2.161280632019043, -0.5212080478668213, 1.3680319786071777, 0.9350833296775818, -0.35421350598335266, -0.29675745964050293, 0.46621641516685486, 0.42511287331581116, -1.7418811321258545, 0.5844271779060364, 1.4949145317077637, 0.5855233669281006, -0.9889436364173889, 0.17115971446037292, -0.3377799391746521, -0.32194650173187256, 2.1021811962127686, 0.6560824513435364, 0.32418370246887207, 0.805677056312561, 0.3706134259700775, -0.1404104232788086, 0.8007745742797852, -0.4845442473888397, -1.5331073999404907, -0.1409013271331787, 0.6424245834350586, -3.1964635848999023, 0.905372142791748, -2.352262496948242, 0.22453902661800385, -0.29281556606292725, -1.0676289796829224, -1.5495996475219727, -3.896226167678833, -0.7913608551025391, -0.24978533387184143, -0.2828074097633362, -1.4911994934082031, 0.804541826248169, -2.2421875, 0.3969973027706146, -0.9141940474510193, 0.7396568655967712, -1.4178142547607422, 0.6994047164916992, -0.5605233311653137, 1.7373027801513672, 0.8965834379196167, -1.1027870178222656, -1.4498172998428345, -0.36260056495666504, 0.4249115586280823, -0.7116239666938782, -0.10205098241567612, 0.06225715950131416, 0.5429210066795349, 0.47103971242904663, 0.6204643249511719, -0.059764206409454346, -0.5477146506309509, 0.27565062046051025, -0.3054565191268921, 0.11950603872537613, -0.20506109297275543, -0.863339900970459, -0.12137585878372192, 0.3587343692779541, -2.2950165271759033, -2.8408923149108887, 0.7456952333450317, 0.0940590351819992, -0.13416163623332977, -1.7121689319610596, -0.27547788619995117, -2.41143536567688, -0.20388464629650116, -1.5264025926589966, -1.850390911102295, -0.44467949867248535, -1.5100204944610596, 0.7187929153442383, 0.8029477000236511, -0.902616024017334, -0.19126202166080475, -0.4934217929840088, 0.9080184102058411, 0.6792067885398865, 0.7184433937072754, -1.239112377166748, 1.0587786436080933, 1.6883106231689453, 2.1335225105285645, 0.8855770230293274, 0.14063848555088043, 0.05956316739320755, -1.5308811664581299, 0.3233618140220642, 2.316990852355957, 0.9029421210289001, 0.2845497727394104, 0.17225827276706696, 2.412111282348633, -2.4520084857940674, -0.39004871249198914, 0.3495335578918457, 1.9404553174972534, 1.112766981124878, -0.1374126523733139, 0.05750873684883118, 0.6209696531295776, 2.1196823120117188, 0.9416446685791016, 0.7545741200447083, -0.7186650633811951, -0.30354025959968567, -0.12983737885951996, -0.3479558825492859, -0.5053147077560425, 0.561817467212677, 1.2112421989440918, -0.8297162055969238, -1.6468075513839722, 0.8586704134941101, -0.2966328561306, -0.8828854560852051, -0.0692618191242218, 0.12903612852096558, 0.9537205100059509, 0.10208742320537567, -0.4780706763267517, -0.7586849927902222, 0.9053446054458618, 0.19409126043319702, 0.6091173887252808, -1.0646982192993164, -1.8123146295547485, -0.876111626625061, -0.6288201212882996, 1.3087379932403564, 1.0410950183868408, 1.9426004886627197, 0.021497471258044243, -0.7324985265731812, 0.8461241126060486, -0.17316287755966187, 1.168382167816162, 0.28029561042785645, -0.6253499984741211, -1.758489966392517, 0.5816355347633362, 1.2111235857009888, 0.31682854890823364, -0.48266369104385376, 0.9782951474189758, -0.432634174823761, -3.097053050994873, 1.3585686683654785, -0.9051048159599304, 0.8760268092155457, 1.9688657522201538, -0.08106843382120132, 1.6599565744400024, 0.4897374212741852, 0.445231556892395, -0.5419362187385559, -0.05895434692502022, 0.9639250040054321, -1.4121873378753662, -2.2373530864715576, 0.6400281190872192, 0.27297529578208923, -0.45099586248397827, -0.16136156022548676, -0.08553282171487808, 1.1329705715179443, -0.770408034324646, -0.6806983947753906, 1.9733790159225464, 2.077754020690918, 0.6142711043357849, 1.4661167860031128, 0.6900113821029663, 0.06224346533417702, 0.8657686710357666, -1.197124719619751, -1.1327722072601318, -0.6487870812416077, 1.0379376411437988, -1.313539981842041, -1.0301581621170044, -0.1122298464179039, 0.6605745553970337, -0.35407620668411255, -0.5674482583999634, 1.170709490776062, -0.6667866706848145, -0.7982438206672668, 0.589609682559967, -0.9560099840164185, 0.5375210642814636, -1.6047780513763428, 0.710608959197998, 1.6969901323318481, -0.4132808744907379, -0.3527933657169342, 0.988883376121521, 0.7777732610702515, 0.6008942127227783, 1.571765661239624, 0.6760148406028748, -1.579431414604187, -0.18850205838680267, 1.8571230173110962, -0.5317149758338928, 0.5280290842056274, -0.6250693202018738, -2.4548697471618652, 0.8136405944824219, 0.8845059275627136, -0.2758018672466278, -1.0092575550079346, 1.1871118545532227, -0.2760084867477417, 2.011159658432007, 0.5782815217971802, 0.22117215394973755, -0.42132750153541565, 2.074479579925537, -1.1368016004562378, 1.7597887516021729, 1.261254906654358, -0.609136700630188, 0.9381888508796692, 1.3467345237731934, -0.02097092941403389, 1.0734305381774902, 0.1371193826198578, -1.4822494983673096, 0.22336305677890778, -0.8297205567359924, 0.4881899952888489, -0.3484337329864502, 0.3466111719608307, -0.272133469581604], [-0.10048434138298035, 0.6312857866287231, -1.117490530014038, 0.45965853333473206, 1.5141583681106567, -0.45089641213417053, 0.28963613510131836, 1.3724901676177979, -0.7312064170837402, 0.8711735606193542, -0.41820287704467773, 0.12247325479984283, 0.9062507152557373, 0.3765687942504883, -0.22235718369483948, 0.7922371029853821, -2.7074220180511475, 0.847346842288971, 0.3166843354701996, 0.8268382549285889, 0.5220681428909302, -0.24152298271656036, 1.8750863075256348, -1.3708257675170898, 0.3210158944129944, 1.4091225862503052, 1.4673068523406982, 0.7653745412826538, -1.545363426208496, 0.05568163841962814, -2.2662250995635986, -0.76691734790802, 0.008634643629193306, 0.7436323165893555, -1.1191025972366333, 1.3694297075271606, 0.6935096979141235, 1.0244224071502686, -0.9385092258453369, 0.304889053106308, -0.23609355092048645, 0.549612283706665, 0.4791038930416107, 1.5699328184127808, 0.7837139368057251, -1.2037551403045654, -0.7637418508529663, 1.2307685613632202, -0.8112140893936157, -0.39330947399139404, 1.3703808784484863, 0.9680233001708984, 0.9084290266036987, 1.066515564918518, 1.698602557182312, 0.723260223865509, -0.7663519382476807, -1.3003928661346436, 1.369118094444275, -0.6950818300247192, -0.7257629036903381, 0.41516730189323425, -0.48727601766586304, 0.9037724733352661, -0.8494501113891602, 0.9348216652870178, -0.8952597379684448, 0.5148161053657532, -0.4048161506652832, -0.15142150223255157, -0.5436729788780212, 0.07810640335083008, 1.2210332155227661, 0.8907598853111267, -1.2860993146896362, -0.6588321924209595, 0.004997162614017725, -0.8425744771957397, -0.3708166182041168, -1.1756682395935059, 1.119681715965271, -0.8009324073791504, -0.3036709129810333, 2.3729910850524902, 0.5132942795753479, -0.9828540682792664, 0.43158605694770813, 0.8243287205696106, 0.2594469487667084, -0.5764084458351135, -1.1101467609405518, 0.3305244743824005, 2.7746074199676514, 1.0078891515731812, -0.343019962310791, 1.5216456651687622, 0.6128119230270386, 0.042917560786008835, -0.7839566469192505, 1.0280613899230957, -1.1722774505615234, 0.304971307516098, 0.7650266289710999, -0.2461482733488083, 0.9498426914215088, -1.7711161375045776, 1.5171096324920654, 0.3839204013347626, 0.38954219222068787, 0.6660337448120117, -2.3650355339050293, -0.2786528170108795, -0.18158018589019775, 0.9214913249015808, -0.4748092293739319, -0.36615443229675293, 1.1547260284423828, -0.24755804240703583, -0.4886337220668793, -1.577797770500183, -2.533457040786743, -1.5445424318313599, 1.7243417501449585, -0.5350114107131958, 1.5867061614990234, 0.5240296125411987, 1.145512580871582, -0.767236590385437, 0.6958250999450684, -0.6578042507171631, 1.650545358657837, 0.5585473775863647, 0.27156785130500793, -0.14508409798145294, 0.14984440803527832, -1.534422516822815, -0.08959272503852844, 1.4296568632125854, -0.4430558681488037, -2.0590834617614746, -0.5679434537887573, -1.791015863418579, -0.9348161220550537, -0.05073177069425583, 2.648088216781616, -2.314253568649292, -0.44557517766952515, -0.5069398283958435, -0.282953143119812, 0.574138343334198, -1.5480902194976807, -0.5093044638633728, 0.5667886734008789, 0.5419932007789612, -1.8511431217193604, -1.3915778398513794, 0.853768527507782, -0.7595082521438599, 0.01917637512087822, 0.7803535461425781, 0.9464355111122131, -0.44281619787216187, -0.1923358142375946, 1.2566736936569214, -1.4274758100509644, 0.5787079334259033, -0.2150372713804245, -0.20602265000343323, 1.0610698461532593, -1.6615666151046753, -0.06676696985960007, -1.750281572341919, 0.6784622073173523, -0.8606550693511963, 0.5010436773300171, 0.6700686812400818, -0.1687719076871872, -0.35625404119491577, 0.06579983234405518, 1.4407833814620972, -0.2200169712305069, 0.5449047684669495, -1.6604384183883667, -1.5005240440368652, 0.5453961491584778, -2.581108331680298, -0.43739742040634155, -0.569159984588623, 2.5382933616638184, 0.4216025769710541, -0.2981623709201813, -0.5763013958930969, -1.3563427925109863, -1.6827762126922607, 0.34869590401649475, -0.12669719755649567, 0.4948967397212982, 1.8656065464019775, -1.1402523517608643, -0.8668004274368286, -0.5450940132141113, -0.8628907203674316, 0.07385916262865067, -1.1687060594558716, -0.48973846435546875, 0.27892163395881653, 0.021820727735757828, -0.4666302800178528, -0.6242226362228394, -1.6542185544967651, 0.09020104259252548, -0.380920946598053, -0.25080499053001404, -0.9947187900543213, -0.055520638823509216, 0.08998537808656693, -1.0723931789398193, 0.06741293519735336, 0.32020169496536255, -0.33528056740760803, -0.07233422249555588, 1.7383627891540527, 0.09661050885915756, -0.7543095946311951, 0.9133965969085693, 0.7866154909133911, -1.1178529262542725, 1.3372951745986938, 1.9633857011795044, -0.21374210715293884, 1.574924111366272, -0.01721401885151863, -0.292041540145874, 0.3118898868560791, 1.5604119300842285, -0.37229835987091064, 0.2134297788143158, -0.10199937224388123, -0.4055689871311188, 1.0123203992843628, -0.11135430634021759, 0.46103954315185547, 1.067711353302002, 1.287386178970337, 0.8849565982818604, 0.6178920269012451, -0.14992591738700867, 0.0023754790890961885, -2.268718957901001, 0.537877082824707, 0.2592393159866333, -0.6309120655059814, -1.2489972114562988, -1.0023998022079468, -0.8574260473251343, 0.38251009583473206, 0.13872745633125305, -0.8827098608016968, -0.6433238983154297, 0.027913620695471764, 0.5394642353057861, -0.5565985441207886, 0.02806744910776615, -1.699937105178833, 0.6939929127693176, -1.4642455577850342, -1.8285787105560303, 0.657523512840271, -0.4076334536075592, -2.915893793106079, 0.17377431690692902, -2.230229139328003, 0.0041972273029387, -1.6716046333312988, 0.13609841465950012, -1.2080206871032715, -0.8399279713630676, 0.01535735558718443, -0.29222097992897034, -0.4740547239780426, 0.5241971611976624, 2.3865416049957275, 0.07967723906040192, -0.950590968132019, -0.6530566215515137, 1.439168095588684, 0.6874455213546753, 1.4894790649414062, -1.1183933019638062, -0.8098204731941223, 0.4852680265903473, 0.7890388369560242, 0.28821104764938354, -0.3362807333469391, -1.0683956146240234, 2.0320520401000977, 0.28919288516044617, 0.0035153147764503956, 0.4120068848133087, -0.012461566366255283, -1.1672579050064087, 0.9751840829849243, -1.5486730337142944, 2.015981674194336, -1.6016134023666382, -0.866662323474884, 1.2454578876495361, 0.8833805322647095, -0.15065167844295502, -0.5283897519111633, 1.1576889753341675, 0.4883594512939453, 1.260545253753662, 0.6697045564651489, -0.9290670156478882, -0.40990495681762695, -0.09507308155298233, 0.6702687740325928, -1.3744664192199707, -0.09106189012527466, -0.8742060661315918, 0.17713072896003723, -0.9291335940361023, 0.8713341355323792, 0.2679034471511841, -0.860794186592102, -0.24318331480026245, -0.5990354418754578, -0.26682618260383606, 0.6212966442108154, -1.596465826034546, -1.014683485031128, 1.292007565498352, 0.2911216914653778, -1.6944655179977417, 3.3140804767608643, 0.1098015233874321, 0.12866376340389252, 0.4604484438896179, 0.2144363820552826, -1.0040360689163208, -1.7126963138580322, 0.36702218651771545, -0.2251829355955124, 0.22166182100772858, -0.7393703460693359, -0.9264408349990845, 0.9738955497741699, 0.6859251260757446, 1.0201752185821533, 0.47051164507865906, 0.9957782626152039, -0.8369003534317017, 0.24386264383792877, -0.7354035377502441, 2.1395132541656494, 1.9791595935821533, 0.060050174593925476, 1.6029900312423706, 0.011454962193965912, 1.1152204275131226, -0.032520998269319534, -0.3782619535923004, -1.9365445375442505, -0.7078560590744019, -2.109562397003174, -0.700381875038147, -0.5456488728523254, 0.8368417620658875, 0.03870762512087822, -0.24487930536270142, 0.611706554889679, 2.7446601390838623, 3.29479718208313, -1.7768802642822266, -0.9793817400932312, 3.3210582733154297, -1.1461330652236938, -0.08412061631679535, -0.09863763302564621, -1.2672665119171143, -1.1466654539108276, -0.3251189589500427, 1.8186231851577759, 0.4333285093307495, -0.8210763335227966, -0.8429272770881653, 0.1454714983701706, 2.122182846069336, -0.7170631885528564, -0.6615267395973206, 0.14590956270694733, 0.849522590637207, 1.1086124181747437, 0.06466248631477356, 1.5044341087341309, -0.0402311235666275, -1.1374918222427368, 1.5147998332977295, -0.7903887629508972, 0.1975165456533432, 0.37028977274894714, 1.0969460010528564, -0.16268092393875122, -0.057518646121025085, 0.11913946270942688, -0.9764890670776367, 0.621625542640686, 1.158658504486084, -0.1163153424859047, 0.6965263485908508, 0.4806559681892395, 0.013329793699085712, 0.5329721570014954, 0.5797207951545715, -0.31815192103385925, -0.07944968342781067, -0.29367244243621826, 1.240659475326538, 1.4001765251159668, 0.45430898666381836, -0.4194868803024292, -0.11189602315425873, -0.31459859013557434, 1.1792651414871216, -1.5675339698791504, -2.223510980606079, 0.8405710458755493, -3.3754899501800537, 1.2049723863601685, 1.7388677597045898, 0.5680596828460693, -2.304788827896118, -0.2997109293937683, 0.022847570478916168, -0.10775593668222427, -0.9038378596305847, -0.5293824076652527, -0.1675422042608261, -0.5936856269836426, -0.2623555660247803, 0.5807028412818909, 1.2499843835830688, -1.229172706604004, 0.4571942985057831, -0.5188551545143127, 0.08921944350004196, -0.6824758052825928, 1.3743634223937988, -0.35855191946029663, -1.0657050609588623, 2.208832025527954, -0.48623549938201904, -0.9744909405708313, 0.17088064551353455, -1.2093437910079956, 0.04223055765032768, 0.5343993902206421, -1.184352159500122, 0.6685223579406738, 1.2961570024490356, 0.25392916798591614, -0.9086196422576904, 2.4646477699279785, 1.5099867582321167, 0.4033451974391937, -2.873335599899292, -1.246544599533081, 1.3011584281921387, -0.9996978044509888, -0.22744207084178925, -0.6511794924736023, -0.8722535967826843, -0.5845990180969238, 0.19735680520534515, 0.8804072737693787, 2.080383062362671, -1.6786718368530273, -1.17096745967865, 0.9951185584068298, 0.05262809619307518, -0.8796088695526123, -1.2498390674591064, 1.1569757461547852, -0.5344884991645813, -0.07790561020374298, -0.12766596674919128, -0.09907041490077972, -0.22636711597442627, -0.3908461630344391, -0.3894064724445343, 1.2349756956100464, 0.26492568850517273, 0.3109002411365509, -0.17326883971691132, 0.41969549655914307, 1.2345486879348755, -1.7478309869766235, 0.6104773879051208, -0.20012420415878296, -1.654998779296875, 0.24600107967853546, 0.5270277261734009, 1.5510181188583374, -0.45214012265205383, -1.322035789489746, -0.1461641639471054, -1.081093430519104, -0.8929650187492371, -1.4276381731033325, -0.9282472133636475, 1.225735068321228, -0.06112055853009224, -0.38353171944618225, -0.7758550643920898, 0.3891281485557556, -0.9965008497238159, -2.046924591064453, 0.8269795775413513, -1.1554348468780518, 0.18652699887752533, -0.91657555103302, 1.1460767984390259, 1.1982172727584839, -1.0324457883834839, -0.6542000770568848, -3.3132059574127197, -0.04223659634590149, -0.18803873658180237, 1.5728238821029663, -2.8693149089813232, -0.2149309515953064, 0.18955111503601074, -0.28925374150276184, 0.01865519769489765, -2.395003318786621, 0.15861791372299194, 1.2364134788513184, 1.3431388139724731, 0.14547336101531982, 0.20811721682548523, 0.32636603713035583, -0.183070108294487, -1.5359359979629517, -0.2963981330394745, 0.5917115807533264, -0.7862377166748047, 1.162471055984497, 0.4136362075805664, 1.008546233177185, -0.8770023584365845, 1.0535918474197388, 0.48762449622154236, -0.9669217467308044, -0.6978588700294495, -0.19743046164512634, 0.7457351684570312, -0.10966554284095764, -0.09800902009010315, 0.5061187744140625, -0.9352359771728516, 1.454159140586853, 0.8887119889259338, -1.3394079208374023, -1.9124295711517334, -1.0049850940704346, -0.6170670986175537, 0.14030510187149048, -1.1747164726257324, 1.0266114473342896, -1.3910421133041382, -0.5877184271812439, -1.3227297067642212, -0.5419284701347351, -0.5128079056739807, 1.2882825136184692, 0.7798209190368652, -0.8307726979255676, 0.2951037585735321, 0.2310912311077118, 0.8964068293571472, 1.992615818977356, 0.3779490292072296, -0.5202632546424866, 0.05819134786725044, -0.4339619278907776, 0.9482624530792236, -0.4904439449310303, -1.3244730234146118, 0.904008686542511, -0.8063592910766602, 0.034053292125463486, -1.2812343835830688, -1.0960603952407837, 0.6508849859237671, 0.3599337041378021, -0.5790631175041199, 0.46007218956947327, 0.5954464077949524, 0.38506269454956055, -1.759024977684021, 0.44609227776527405, 2.489353656768799, 1.8380545377731323, 0.531757116317749, 1.1465157270431519, -0.26421141624450684, 0.03222304582595825, -0.58845055103302, -0.1978822946548462, -1.4294185638427734, 1.9899977445602417, 0.2669256627559662, -1.4543853998184204, -0.1498885303735733, -1.746104121208191, -0.23514971137046814, 0.2495369166135788, -1.0521007776260376, -0.1948380470275879, -0.10541024804115295, -1.0636404752731323, 0.743815541267395, 1.1729625463485718, 0.128798246383667, -0.23477527499198914, -0.8953628540039062, -0.32661712169647217, -1.7167210578918457, -0.08351630717515945, 0.3777358829975128, 0.49619168043136597, 0.7464407086372375, 2.9715347290039062, 1.6540073156356812, -0.4671030640602112, 0.19536910951137543, -1.3473645448684692, 1.9251525402069092, -0.25925812125205994, 0.947036862373352, 0.028006382286548615, -0.7307329773902893, 0.16254809498786926, -1.8110982179641724, 0.13798217475414276, 1.8191003799438477, 0.7912662029266357, 0.634540855884552, -0.6139301061630249, 0.6079358458518982, -0.15808416903018951, -1.5259218215942383, -0.003371636150404811, 0.5180867314338684, 1.3696681261062622, 1.8568986654281616, -1.1812824010849, 1.1339424848556519, 0.09639465063810349, -1.1269419193267822, -0.3096937835216522, -0.3699791431427002, 0.9195258021354675, 1.4224703311920166, 1.5927919149398804, -0.3122338354587555, -1.0622202157974243, -1.0463694334030151, 0.53184574842453, 0.5297380685806274, 1.2255473136901855, 0.13086192309856415, 0.9623289108276367, 0.7238019108772278, -0.3316359519958496, -0.5601242184638977, 0.7319804430007935, 0.4092002213001251, 0.35932907462120056, -0.255036860704422, -0.19134196639060974, 0.2659587264060974, 1.9054558277130127, 0.839955747127533, -1.329908013343811, -0.8255716562271118, -2.1714911460876465, 0.5971948504447937, 1.7757562398910522, 0.03552267700433731, -0.010269839316606522, 1.0490803718566895, -1.113356113433838, -0.8069524168968201, -0.27374061942100525, -0.3428496718406677, 0.5466496348381042, -0.0024334124755114317, -1.014801263809204, 0.48801228404045105, -0.34258317947387695, 0.5165608525276184, 0.7175553441047668, 1.4808119535446167, -0.15599863231182098, -0.8362374305725098, -1.197851300239563, -1.0765644311904907, -0.4818912744522095, 0.9736855030059814, -0.7371348142623901, 1.051111102104187, 0.903041660785675, 0.41309085488319397, -1.4141072034835815, -0.7099222540855408, 1.7408829927444458, -1.5712357759475708, -1.2400965690612793, 0.01971038430929184, 2.3002419471740723, -0.5537874698638916, 0.9248523116111755, 0.5934273600578308, -0.6147714257240295, -0.5308362245559692, 1.0936275720596313, -0.8919321894645691, 0.9867204427719116, -0.6773123741149902, 2.3013947010040283, 1.0010682344436646, 1.3071622848510742, 2.525564670562744, 0.19694913923740387, 0.41271811723709106, -0.2547018527984619, -1.6732898950576782, 0.552139937877655, -1.1814647912979126, -0.16895559430122375, 0.11321323364973068, 0.6501627564430237, 0.04936026781797409, -1.511813998222351, 0.5454338192939758, -1.2465349435806274, 0.41970136761665344, 0.41500526666641235, 1.8681252002716064, -0.0925668478012085, -0.7059066295623779, 0.33525174856185913, -0.05679433047771454, -0.10818663239479065, -1.2411689758300781, -0.9554360508918762, -0.6925362944602966, 1.1039378643035889, 0.9696006178855896, -0.11662785708904266, 0.466484397649765, -1.3512046337127686, 1.067793846130371, 1.27739679813385, 0.7794250249862671, -0.22063614428043365, -0.6439671516418457, 0.08616431057453156, -0.7973747849464417, 0.25588762760162354, 0.45466166734695435, -2.4389355182647705, -0.7966696619987488, 1.137905240058899, 0.6794514060020447, 0.8411228656768799, 1.679307222366333, 0.14400309324264526, -1.6769185066223145, 0.06935561448335648, 0.3838335871696472, -0.4268597960472107, 0.3435685932636261, -0.18206363916397095, 0.1642330139875412, 0.24802440404891968, -0.9919052124023438, -0.0879763513803482, 0.129354327917099, -0.19531303644180298, 0.07552802562713623, 2.077524423599243, -3.9325833320617676, 0.03005479834973812, 0.10802309215068817, -0.347966730594635, -0.15519706904888153, 0.13496731221675873, 0.8681369423866272, 0.1520196944475174, -0.43746647238731384, 0.8437130451202393, -0.39858540892601013, 0.5755692720413208, 0.9419879913330078, -1.0010015964508057, -0.23220203816890717, 0.07054418325424194, 1.5890306234359741, 0.5419027209281921, -0.37390270829200745, 1.6537034511566162, -0.4156060814857483, -0.4938157796859741, 0.6404901146888733, 0.01965995877981186, 1.5566471815109253, 0.07204708456993103, 1.6626487970352173, -0.8000562787055969, -1.6291464567184448, -0.242955282330513, -1.3205478191375732, 0.24100713431835175, -0.2284308820962906, 0.4928339421749115, 0.631583034992218, -0.6017704606056213, -0.12275553494691849, -0.4458644986152649, -0.7219750881195068, 0.7580389976501465, -0.33661088347435, -0.7744387984275818, -0.23538966476917267, 1.3587032556533813, 0.7448732256889343, -0.20449063181877136, 2.1416001319885254, 1.1454049348831177, -0.6458795070648193, 0.2933126986026764, 0.4165588319301605, 0.6660633683204651, -0.060368288308382034, 1.187135934829712, 1.2405487298965454, -0.6069650650024414, -0.2609742283821106, 0.532468855381012, -2.304729461669922, 0.8866580128669739, 0.8414695262908936, 2.108236312866211, -0.2648838758468628, 1.0310258865356445, 0.6527367830276489, 1.6033967733383179, -0.13697077333927155, 1.255048394203186, -0.3260471820831299, -1.5500175952911377, 1.8372150659561157, 1.2291654348373413, 1.0343055725097656, 0.33116060495376587, -0.12549293041229248, -0.5956518054008484, -0.004739915486425161, 0.28769075870513916, 0.31866177916526794, -0.4668259918689728, -0.4302361011505127, -0.09154342859983444, 1.5219159126281738, 0.38639482855796814, -0.5589229464530945, 2.775545120239258, 0.36057254672050476, 0.26386716961860657, -1.0782934427261353, 1.1433813571929932, 2.4627838134765625, 0.48417654633522034, -1.0049049854278564, 0.7016631960868835, 0.015437453985214233, 0.2226921170949936, 0.6655197739601135, 2.1425275802612305, 0.14228063821792603, -0.1825598031282425, -1.1318092346191406, 1.0003787279129028, 1.371457576751709, 0.8607774972915649, 1.3416504859924316, 2.248305559158325, -0.42810356616973877, 0.44124728441238403, 0.4141186773777008, -1.1627413034439087, -1.4229302406311035, 0.33338049054145813, 1.2628947496414185, 0.11232281476259232, -0.7873824834823608, 0.885145902633667, 0.09965718537569046, -0.03100113570690155, 1.6293060779571533, 0.09528373181819916, -1.1113834381103516, 1.5552624464035034, -1.7655507326126099, -1.519739031791687, 2.005446434020996, -0.567873477935791, 0.7010997533798218, -1.4770607948303223, 1.5528788566589355, -1.1885430812835693, -0.5135779976844788, -1.7063944339752197, -1.1030423641204834, 1.111842155456543, -0.6663587689399719, -1.3222715854644775, 0.20557639002799988, 1.5680437088012695, 0.760680079460144, -1.2726068496704102, 0.33918216824531555, 1.937788486480713, 0.37376394867897034, -0.36055809259414673, 0.03373825177550316, -0.8126147389411926, -0.6275745630264282, -0.30343249440193176, -2.440781354904175, 2.2567038536071777, -0.9529503583908081, 0.038141436874866486, -0.544071614742279, -0.7485433220863342, 0.9960415959358215, 0.6016119718551636, 0.5127391815185547, 0.3499196469783783, 2.7348530292510986, -2.0575265884399414, 1.3181706666946411, 0.2951156497001648, 1.3290340900421143, 1.1309210062026978, 1.4675602912902832, -1.439042568206787, -0.3757607042789459, -0.5531801581382751, 0.38731372356414795, 1.292715311050415, -0.5900357365608215, 0.2346363663673401, 0.46602770686149597, 0.029860710725188255, 0.7713201642036438, -0.617031991481781, 0.5666853785514832, -1.1375972032546997, -0.11903142184019089, 0.02312762476503849, 0.506555438041687, -1.8733457326889038, -0.41531825065612793, -2.8983314037323, -0.5511928200721741, 0.27076616883277893, 0.6461912393569946, -1.6459012031555176, 0.2830825448036194, 0.2508528530597687, -0.16401691734790802, 1.1056846380233765, 1.4072141647338867, -0.15055030584335327, 0.8726840019226074, 2.582712411880493, -0.7855818271636963, 0.5090945363044739, 0.3039248585700989, -1.1465785503387451, -0.04198511317372322, -1.2835159301757812, 0.42150065302848816, -1.0023397207260132, 0.18441158533096313, -1.4544134140014648], [0.4815404713153839, 1.2519481182098389, -1.7071046829223633, 0.9021393656730652, 1.0949029922485352, 2.5132057666778564, 0.019551001489162445, 1.7497798204421997, -0.30477941036224365, 0.017763620242476463, 1.9343839883804321, -0.9338040947914124, -0.08698495477437973, -1.7726736068725586, -0.3065930902957916, -1.3992642164230347, -1.9300267696380615, -0.4811769723892212, -0.8409503698348999, -1.004390835762024, 0.44515201449394226, 1.0359514951705933, -0.05973101779818535, 0.3773265480995178, -2.086559534072876, 2.288212776184082, 0.22441324591636658, -1.338939905166626, -1.1624476909637451, 1.1706377267837524, -0.05675675719976425, 1.258776068687439, 1.2467678785324097, 1.1685761213302612, -1.705544352531433, 1.0075268745422363, -1.6718838214874268, 0.29497528076171875, 1.135014295578003, 2.369473934173584, -0.811382532119751, 1.4612619876861572, -0.06791049987077713, -1.189699411392212, -0.23107945919036865, 1.512598991394043, -0.010916076600551605, 1.6750868558883667, 1.1866668462753296, 0.6590838432312012, -0.27127936482429504, 0.013991591520607471, -0.8018734455108643, -0.8991965651512146, -0.6522638201713562, 0.7424116730690002, 0.3091467022895813, -1.3802570104599, -0.007778131403028965, -0.9604949951171875, -0.5737121105194092, -0.08612476289272308, -0.17736651003360748, -0.006050093565136194, -2.1403439044952393, -0.9076470732688904, -0.4252929091453552, -0.44031819701194763, -0.41187527775764465, 0.9730657935142517, -0.640925407409668, -1.4933773279190063, -1.5234172344207764, 0.9251982569694519, 0.2171722948551178, 1.5197010040283203, -0.16062289476394653, 0.8793859481811523, -0.09491077810525894, -0.27902528643608093, -1.6833637952804565, -1.161177635192871, -0.31671130657196045, -0.6962396502494812, -0.4162200391292572, -0.500442385673523, -0.846764326095581, -0.7328670620918274, 0.2415776401758194, 0.5764394998550415, 0.14348681271076202, 2.2430992126464844, -1.3905322551727295, -0.12901636958122253, 0.05724561586976051, 0.07130376249551773, 0.42511382699012756, -0.08726124465465546, 1.2997246980667114, 0.3040385842323303, 0.030664801597595215, -0.27951690554618835, 0.9669761657714844, -1.5472389459609985, -1.3119733333587646, 0.8440999984741211, -1.1817339658737183, -1.435407042503357, -0.050167739391326904, 0.11997094005346298, -0.3796596825122833, 1.85848069190979, -1.4921751022338867, 0.07820302993059158, 0.7563818693161011, 0.09812244027853012, -0.5662902593612671, -1.6625540256500244, 0.19307823479175568, 0.7141492366790771, 1.4709433317184448, 1.3393537998199463, -1.6431583166122437, -0.4317559003829956, 0.5228219032287598, 0.9323135018348694, -0.1770172417163849, 0.30598241090774536, -0.3665235936641693, -0.6897595524787903, -0.9653574228286743, -0.557816743850708, 1.1724997758865356, -1.4685258865356445, 2.0004568099975586, -0.06689045578241348, 0.4889722466468811, -1.4573657512664795, -0.46263960003852844, 0.3185490667819977, 1.0668402910232544, 0.0006318177911452949, 0.39766570925712585, -0.4194571375846863, -0.3178894817829132, 0.18116891384124756, -1.2597464323043823, 0.9940171241760254, -0.31376132369041443, -1.0801105499267578, 1.073423981666565, 0.13658419251441956, -0.9457613825798035, 0.46361902356147766, 0.5751447677612305, -0.08610806614160538, -0.3231048583984375, -0.12937889993190765, -0.6952497959136963, -1.2207095623016357, -0.7107791900634766, -0.22533240914344788, -0.6022534966468811, 1.8225668668746948, 0.20095205307006836, -0.4080248475074768, -0.3807377815246582, -0.524577260017395, 0.48303571343421936, -0.006082612089812756, 0.7769688367843628, 0.23516426980495453, 0.7262193560600281, -0.4569928050041199, -0.3006684184074402, 0.26291996240615845, 0.13377320766448975, 1.2427488565444946, 0.7839074730873108, 0.6183978319168091, -1.1536678075790405, 0.9155555367469788, -0.6763116717338562, 0.09493910521268845, 2.205582618713379, -2.735382080078125, 0.36615312099456787, -1.1902915239334106, 0.04415182024240494, -0.48279985785484314, 0.7367099523544312, -2.1677708625793457, 1.1439154148101807, 0.5993438363075256, -0.8979920148849487, 0.9895158410072327, -0.08056551218032837, 1.527424693107605, -1.1572273969650269, 0.08817736804485321, 0.8785979747772217, 0.6518465280532837, -0.05601269006729126, -0.5729013085365295, 1.04165780544281, 0.30958443880081177, 1.016536831855774, -1.9603415727615356, 0.15446537733078003, -0.9768465757369995, -0.6096271276473999, -0.4761928915977478, 1.3771342039108276, 1.5857863426208496, 0.04489239677786827, -0.873216986656189, 0.4942924976348877, 0.7915919423103333, -0.6657302379608154, 2.080087661743164, 0.9560622572898865, 0.29298388957977295, -0.46594277024269104, -0.041426874697208405, 0.039466168731451035, -0.4817115068435669, -0.549293577671051, -1.136717438697815, -0.4922350347042084, 0.18812397122383118, 0.5917759537696838, 0.9800450801849365, -0.543195366859436, 0.6726483106613159, -0.557598888874054, 0.7532561421394348, -0.41913163661956787, 0.2715418338775635, 0.06809824705123901, -0.012937917374074459, 1.5102035999298096, 0.0790407806634903, 0.10584540665149689, -0.6697403192520142, 0.05889160931110382, -0.10193956643342972, -2.584213972091675, 0.5325488448143005, 0.3389357626438141, 0.09541645646095276, -1.1879092454910278, -0.0660470724105835, -0.5904738306999207, -0.7589755654335022, -0.19103799760341644, -0.018924003466963768, 0.2618533670902252, -1.1221219301223755, -1.8927704095840454, 0.7863505482673645, 1.2392635345458984, 0.1302453726530075, -0.9376247525215149, -0.9454646110534668, 0.9585558176040649, 0.33121833205223083, 1.705108404159546, 0.12176220864057541, -0.42409729957580566, -0.7407501339912415, -0.22367489337921143, -1.4935612678527832, -1.4524003267288208, -0.7208362221717834, 0.458870530128479, 2.5557191371917725, 0.36108914017677307, -1.0822569131851196, -0.2598772943019867, 0.022350028157234192, -1.3215128183364868, 0.10719428211450577, 1.0953963994979858, 0.6212549209594727, 0.28794461488723755, -1.0909610986709595, -0.3352949917316437, 0.8336194753646851, -0.9249584078788757, 2.281696081161499, 0.45489537715911865, -1.1057096719741821, 0.11579754948616028, -1.022833228111267, 1.3098299503326416, -0.02115447446703911, 2.0463778972625732, 0.07558200508356094, -0.41781285405158997, -0.6380380392074585, 1.069593071937561, -0.5247069597244263, 0.6583989858627319, -1.883109450340271, -1.4443849325180054, -1.0770535469055176, 0.5793823599815369, -0.6062857508659363, 0.28546881675720215, -0.4297749996185303, 0.41717466711997986, 0.7061222195625305, 1.1146785020828247, -0.031114740297198296, 0.6155348420143127, 0.8452877998352051, -0.47874805331230164, 0.7931323647499084, -1.3042110204696655, 0.6303396224975586, 1.2629321813583374, 0.03307485580444336, -0.8643143773078918, -0.7837831974029541, -0.3195403516292572, -0.5110741853713989, -0.0926692932844162, -1.2713878154754639, -0.2535076439380646, -0.9085592031478882, -0.20651765167713165, 2.9359936714172363, 0.9064485430717468, 0.30661600828170776, 0.016253549605607986, -0.9369388222694397, -0.6083712577819824, -1.5054012537002563, -1.571940541267395, -1.2603784799575806, 0.17007403075695038, 0.09634225070476532, 0.06819178909063339, 0.4181550145149231, 1.7694958448410034, -1.1209784746170044, -0.7922064661979675, 0.4485195577144623, -0.9132304787635803, -1.3800626993179321, -0.7541491985321045, -0.5275337100028992, 0.040404945611953735, 1.3692158460617065, -1.4238754510879517, 0.5242577195167542, 0.17322948575019836, -0.3108522593975067, 0.23463931679725647, -0.15720511972904205, -0.023188147693872452, 0.6108651161193848, -0.12109910696744919, 0.16017942130565643, -0.35137665271759033, -2.459104537963867, 1.0094789266586304, 0.5088630318641663, 1.2735085487365723, -0.5251449346542358, 0.24613356590270996, -0.17639079689979553, -0.3405955135822296, 1.211858868598938, 0.8298792243003845, 1.9384706020355225, -1.6199320554733276, -0.08744063973426819, 1.4712276458740234, 0.4070928692817688, 0.047782331705093384, -0.9015166759490967, 0.6990777254104614, -0.11486899107694626, 0.56893390417099, 0.8997344970703125, -0.7424177527427673, -0.5921481847763062, -1.2231520414352417, -0.8997136950492859, 0.6595072150230408, -1.1553105115890503, -1.3027478456497192, 1.9269903898239136, -0.7910497188568115, -0.8355317711830139, 0.18489161133766174, 0.30371734499931335, 0.25633126497268677, 2.7621428966522217, 0.7654528617858887, 0.4392365515232086, 0.8628950119018555, -0.020620621740818024, 0.7204474806785583, 0.7202025055885315, 0.7788977026939392, -1.1741795539855957, -1.4384135007858276, 0.8564755916595459, 0.9140225648880005, -1.0993731021881104, -1.539002537727356, -0.5850714445114136, -1.1275429725646973, 0.4174307584762573, -0.7390894889831543, 0.2779061496257782, -0.637971043586731, -1.5262171030044556, -0.042716898024082184, -0.9053245782852173, -1.9374120235443115, 0.5979899168014526, -1.7738580703735352, -0.08589522540569305, 0.08972189575433731, 0.4537680447101593, 1.4702931642532349, 1.3623746633529663, 0.15282249450683594, 0.30177953839302063, 0.15180999040603638, -0.25567135214805603, -0.696260929107666, 2.255960464477539, -1.037285327911377, 1.0616155862808228, 0.5328091979026794, 0.5423029065132141, 0.2439412772655487, -0.6333895921707153, 0.155769482254982, 1.0335019826889038, -0.7032971382141113, 0.4208817183971405, 1.343483328819275, 1.6171832084655762, -0.457669734954834, 0.4568400979042053, 0.2830018103122711, -0.19004569947719574, 2.48104190826416, 0.9566069841384888, -0.4769534170627594, -1.6679863929748535, -1.592307448387146, -0.9671544432640076, -1.127949833869934, -0.0148192523047328, 0.9214485287666321, 0.2990133762359619, 0.6131492853164673, 0.8181067705154419, -0.5054244995117188, 1.1488666534423828, -0.9308395385742188, -0.4905526638031006, 1.1513770818710327, 0.35430341958999634, -0.33566418290138245, 0.21511775255203247, 0.642968475818634, 0.29470008611679077, -0.2705717384815216, -0.25648003816604614, 0.7969865798950195, 0.41111695766448975, 0.35858455300331116, -0.803901195526123, 0.16660729050636292, 0.25200384855270386, 1.6354283094406128, -1.3262007236480713, 0.8177682161331177, -0.9318950176239014, -1.3993972539901733, 0.139474019408226, 1.1434303522109985, -0.39763158559799194, 0.05703386291861534, 1.906583547592163, 1.1735368967056274, -1.4727364778518677, -1.4486955404281616, 2.408015727996826, -1.0906691551208496, -2.4504587650299072, 3.109621286392212, 0.08517013490200043, -0.14574432373046875, 0.09759481251239777, 1.5400454998016357, -0.3473341464996338, 0.6500054001808167, -0.8405001759529114, 0.8238471746444702, -0.16305558383464813, -1.6030043363571167, -0.39362233877182007, 1.4181164503097534, -1.041935682296753, -1.075278401374817, -1.5475494861602783, -0.4542483985424042, 0.26404204964637756, 0.3786521255970001, 0.7918134927749634, -0.19435003399848938, 0.5657078623771667, -1.6285568475723267, 0.8683974146842957, 0.4194694459438324, 1.5258749723434448, -0.9465329647064209, 1.4841861724853516, -1.644429087638855, -0.5542768239974976, 1.0536044836044312, -0.08542978763580322, -0.9631413817405701, 0.04227663576602936, -0.3918357491493225, 2.599620819091797, -1.3333535194396973, -0.9171244502067566, 0.7202606797218323, -1.409766674041748, 1.1604739427566528, 1.1237848997116089, -0.7708011269569397, 0.1645166128873825, -1.111412525177002, 1.0819848775863647, 0.07053154706954956, -0.4869726002216339, 0.15010392665863037, -0.03741466626524925, -1.3752537965774536, 0.296511173248291, -0.29202792048454285, 0.41702595353126526, 1.2131637334823608, 1.4431389570236206, -0.8740655183792114, -0.3477787673473358, 0.08800559490919113, 1.3958081007003784, 1.0749256610870361, 0.6791213750839233, 1.1037052869796753, -1.1819820404052734, -0.17112156748771667, 0.8871099352836609, 1.709324836730957, 1.0477004051208496, 0.4197276532649994, 0.3509698808193207, 1.2552838325500488, 0.794935405254364, 2.1815004348754883, -0.407365083694458, -0.9137227535247803, 0.23985205590724945, -0.21932615339756012, 0.4796047508716583, 0.8475027084350586, -2.512728214263916, 1.0975399017333984, 1.3554749488830566, -0.06808225065469742, -0.09887616336345673, 1.264427900314331, -0.15113496780395508, 1.2286174297332764, -1.2924166917800903, 1.4037859439849854, -0.9525843858718872, 0.2332632839679718, 0.5566365122795105, -0.971146285533905, -0.444430947303772, -0.8096261620521545, 1.1781885623931885, -0.5358083248138428, 0.019451728090643883, -1.6670128107070923, 1.2054047584533691, 0.7924144864082336, -0.24249841272830963, 0.16708947718143463, -0.5374874472618103, 0.9848830103874207, 1.1232584714889526, -1.1580387353897095, 0.06911415606737137, 0.2084972858428955, -0.7156503200531006, -0.1446533352136612, -0.34646815061569214, 0.36130526661872864, -2.6475791931152344, 0.0321061946451664, -0.725045919418335, -0.00405395682901144, 0.019893381744623184, -1.2400832176208496, -1.2112681865692139, 0.08939779549837112, 1.78633451461792, -0.885696530342102, -0.46165740489959717, 0.6300153732299805, -0.5586656332015991, 0.8915669918060303, 2.596837282180786, -1.5901800394058228, 1.2044323682785034, 1.1030211448669434, 0.29649069905281067, -0.16879072785377502, -1.0053497552871704, -1.1661962270736694, -0.04674305394291878, -0.32122835516929626, 0.4434809684753418, 0.5480976700782776, 2.1900501251220703, -1.043408989906311, -3.0582211017608643, -1.9004385471343994, -0.6673125624656677, -0.8954945802688599, -0.9612985849380493, 0.614956259727478, 0.30906054377555847, -1.943005919456482, 0.2516617476940155, 1.0223907232284546, 0.297336608171463, 0.981396496295929, 0.22461505234241486, 1.1669965982437134, 1.6221572160720825, 1.324223279953003, 0.596110999584198, -0.723544716835022, -0.46911996603012085, 1.3451460599899292, -0.4471210241317749, 0.22363635897636414, -0.23659686744213104, -0.3310769200325012, 1.4744983911514282, -0.7911101579666138, -1.3808571100234985, -0.5165000557899475, -1.3160635232925415, 0.8184518814086914, -0.47631552815437317, 0.4427363872528076, 0.2338142842054367, 0.951017439365387, 1.6125552654266357, 1.8692961931228638, -0.2755211293697357, 1.454248309135437, 0.9457743167877197, -0.7009238004684448, 0.2521379590034485, -0.18082229793071747, 1.9369146823883057, -0.854566216468811, 0.210865780711174, 0.7836018204689026, -0.7078321576118469, 2.443852186203003, 0.8291882872581482, 1.9943753480911255, 2.009643793106079, 0.9966618418693542, 2.440304756164551, 0.40221133828163147, -0.009733923710882664, -1.1077244281768799, -1.985095500946045, -1.1166801452636719, 0.061444420367479324, -1.147350549697876, -1.248158574104309, -0.8960821032524109, 0.3143071234226227, -1.0660450458526611, -0.6002321839332581, -0.16770631074905396, -1.6726130247116089, -0.5394473075866699, -0.042613375931978226, -0.8282470107078552, -0.2983374297618866, -0.4366324841976166, -0.11494401097297668, 0.951934278011322, -0.3384787142276764, -0.21671649813652039, -1.555020809173584, -1.1089811325073242, 1.1443073749542236, -1.9448986053466797, -2.772374153137207, 0.11960029602050781, -0.4740871787071228, -1.3842520713806152, -0.00437661400064826, -0.3535996079444885, -0.6383545398712158, -1.0458228588104248, 0.38997283577919006, -0.9663235545158386, 1.6290415525436401, -0.06729590892791748, -1.0435094833374023, -1.112241506576538, 0.16656875610351562, -0.05144455283880234, 0.6913318634033203, 0.530093789100647, -0.014192361384630203, 1.7154309749603271, -0.7231617569923401, -1.0003221035003662, 1.0046955347061157, -0.9681729078292847, -0.31160926818847656, 0.29174473881721497, 0.4028862416744232, -0.9168326258659363, -0.23131154477596283, -1.0829741954803467, 2.2521870136260986, 0.5130441784858704, -0.10257089883089066, 0.8702002763748169, -0.6533187031745911, -1.0233489274978638, 0.8868779540061951, -0.21097467839717865, 0.47325822710990906, -0.36500072479248047, 0.11994826048612595, 0.021773137152194977, -0.8166739344596863, -0.17087769508361816, 1.689822793006897, -0.13978423178195953, -1.1254128217697144, 1.2660962343215942, 0.9904685616493225, 0.7309476733207703, -0.02177818864583969, -0.17615769803524017, 2.1698975563049316, -0.27070051431655884, -0.5720077753067017, -0.3944917619228363, -0.6264369487762451, 0.898639440536499, -0.4205612540245056, 0.19368843734264374, 1.0213420391082764, 0.5912036895751953, -0.3254888951778412, -2.0144686698913574, -0.2603895664215088, 0.29346928000450134, 0.3573271632194519, 0.3560590147972107, -1.275853157043457, 1.3875477313995361, -0.5121450424194336, -0.02429494634270668, 0.47196632623672485, -0.9103574156761169, 0.15945234894752502, -0.2375919222831726, 0.5692058801651001, -1.3287416696548462, 0.1376982480287552, -0.161724254488945, 1.0680108070373535, 1.2958954572677612, 0.2153172791004181, -0.06338869035243988, 0.6827892065048218, 0.16988316178321838, 0.2895677983760834, -0.16274382174015045, -0.8990263938903809, -1.8483221530914307, -0.0025735467206686735, -0.38540220260620117, -2.016202926635742, 0.3869522511959076, 2.447559118270874, -1.0831871032714844, 0.2691541314125061, -0.7548497915267944, -0.14115947484970093, 1.1407662630081177, -0.6228891015052795, 0.08194166421890259, 0.7946961522102356, -0.5194925665855408, -0.25201091170310974, 0.00794400554150343, -0.7548878192901611, -0.4834124445915222, -0.4368528723716736, 0.28075888752937317, -0.6817398071289062, -0.021850889548659325, 0.16462737321853638, -1.5367597341537476, 0.5450078248977661, -0.05197827145457268, 0.6519944071769714, 0.2989661693572998, -0.41181236505508423, -1.532776951789856, -1.5526138544082642, -0.6408877372741699, -1.0864700078964233, -0.24284492433071136, -0.4551093280315399, -1.1033875942230225, 1.3089444637298584, -1.230368733406067, 0.6274061799049377, -1.035213828086853, 0.6366097927093506, 0.41448426246643066, 0.300001323223114, -0.4725719690322876, 0.9680396318435669, -0.15838974714279175, 0.8761118650436401, -0.055753495544195175, -0.2843948304653168, -0.24189263582229614, 0.09493475407361984, 1.9201068878173828, -0.8131670951843262, -1.0172169208526611, 0.20179811120033264, 0.14419808983802795, -1.1169712543487549, 0.9420479536056519, -0.9881681799888611, 0.7546184062957764, -2.0198421478271484, 0.40800341963768005, 0.19997930526733398, 0.08230404555797577, 0.2625635862350464, 1.017050862312317, 0.17341814935207367, 1.5645155906677246, 1.12509286403656, -0.9312533140182495, 0.567112922668457, -2.1579391956329346, -0.8634819388389587, -0.5326293110847473, -0.7863562107086182, 0.49691227078437805, -0.22395537793636322, -1.0218963623046875, 0.38339900970458984, 0.2857159674167633, 0.741749107837677, -0.6316828727722168, 1.4494937658309937, 0.02602059207856655, -0.935337245464325, -0.6712173223495483, 0.4192802906036377, -0.7858818173408508, 1.3554407358169556, -0.8633808493614197, -0.5956851243972778, -0.28742316365242004, 0.7839162945747375, -2.596200704574585, -0.43133997917175293, 0.3077225089073181, 0.7679370045661926, -0.2736225724220276, 0.9838430881500244, -0.16010703146457672, 0.6427459120750427, 2.622889280319214, 1.2226368188858032, 1.0892724990844727, -0.49128496646881104, 2.024872064590454, -1.2379909753799438, 0.0022909697145223618, -0.2590669095516205, 0.2543044090270996, -0.2705484628677368, -1.257796287536621, 1.3723753690719604, -0.8807249069213867, -0.1432945430278778, 1.2694588899612427, -0.3859589695930481, 0.9692203402519226, 0.5539146661758423, -0.0015486497431993484, 0.6312980055809021, -0.18113933503627777, 0.4850906431674957, -1.190022349357605, -0.9474531412124634, -1.072664737701416, -0.7405453324317932, 1.118566870689392, -0.028842909261584282, 0.6330779790878296, 0.23197729885578156, -0.48596248030662537, -0.9705802798271179, 0.3810928165912628, 0.11911581456661224, -1.7416683435440063, -0.34062880277633667, 2.664910078048706, 0.27443745732307434, -0.3710366487503052, 0.5917118191719055, 0.10569123178720474, 1.209831714630127, 0.9760259985923767, 2.0371177196502686, 0.14707110822200775, 1.321009874343872, 0.3406166732311249, 0.9545531868934631, -0.14337502419948578, -0.1381177306175232, -0.4224984049797058, -1.0203436613082886, -0.11513285338878632, 0.31793612241744995, -0.5778910517692566, -0.20522259175777435, -0.9704160094261169, -0.9038091897964478, -0.07731028646230698, 1.260514259338379, 0.8998445272445679, 0.845837414264679, -0.1913229525089264, -0.3370513617992401, 0.36909082531929016, 0.3019549250602722, -0.06495172530412674, -0.19038033485412598, -0.9140030145645142, -0.2607358694076538, -0.9677917957305908, -0.4718760848045349, 0.9600300192832947, 0.9061073660850525, -0.7338801622390747, 0.8680943846702576, -1.1789559125900269, 1.2698771953582764, -1.460881233215332, 0.16522182524204254, -0.5657860040664673, -1.027328372001648, -0.7279413938522339, -0.4381319284439087, -0.10759903490543365, -1.3570115566253662, 0.7837005853652954, -0.12063182145357132, 1.7949589490890503, 1.309510588645935], [-2.3709981441497803, 0.908689558506012, 1.072303056716919, 0.3961647152900696, -0.7480794191360474, 1.2261385917663574, -0.7073268890380859, -0.7237256765365601, -1.0366978645324707, 0.5484720468521118, -0.49477702379226685, -0.7142917513847351, -0.18106494843959808, -1.2124532461166382, -0.36304858326911926, 0.2757182717323303, 2.0627238750457764, 0.6369825601577759, 1.1298857927322388, -0.2865500748157501, 0.5142970681190491, 2.531024932861328, -1.139177680015564, 0.7208728194236755, 0.23223379254341125, 0.8537010550498962, -0.2410484254360199, -0.7058131098747253, 1.0311185121536255, 1.1286876201629639, 0.3007449507713318, 0.6526280641555786, -1.4962304830551147, 1.2574974298477173, 0.2037186175584793, -1.4968253374099731, 0.3431817591190338, 0.7420434951782227, 1.6588687896728516, -0.7934517860412598, 0.3782360553741455, 0.8190475702285767, 1.215261697769165, -1.2764405012130737, 0.7378790378570557, 3.0607330799102783, 1.5186913013458252, 1.7510849237442017, -1.8212506771087646, -0.13535137474536896, 1.193548560142517, 0.3454291522502899, -0.4774699807167053, -1.1122277975082397, -0.43237438797950745, -0.09466474503278732, -0.15414196252822876, -0.3494519293308258, 0.17527465522289276, 2.133162260055542, -0.5879455208778381, -0.09294421225786209, 0.5782960653305054, 0.8782880902290344, 0.649337887763977, -1.466570258140564, 3.127211809158325, -0.3954164981842041, 0.6302188634872437, -0.4951027035713196, 0.9678700566291809, -0.5598928928375244, -0.2906546890735626, 0.7147113680839539, 1.2406302690505981, 1.9021339416503906, 0.6707162857055664, -0.19217701256275177, -0.8676550388336182, 0.0313909687101841, -2.560169219970703, 0.5228689908981323, 1.4557682275772095, 0.531090497970581, -2.2177486419677734, 0.9148890972137451, -1.5666476488113403, 0.5809869170188904, 0.7467378973960876, 1.7661938667297363, -0.734458327293396, 0.972382128238678, 2.7136874198913574, -0.7439387440681458, 1.9628851413726807, 1.4038496017456055, -0.4556754529476166, -1.6188197135925293, 1.1314518451690674, -1.1566238403320312, 0.8969070315361023, -1.7854427099227905, 1.3461353778839111, -0.34875383973121643, -0.6201717257499695, -0.8527133464813232, 1.1185929775238037, 0.6267769932746887, -0.11264318227767944, 0.42953628301620483, -0.15550145506858826, -0.6848852038383484, -0.0010087667033076286, -0.9859567880630493, -1.4460095167160034, -0.33300116658210754, 2.025918960571289, -1.0033812522888184, -1.3356523513793945, 1.0978881120681763, -0.2032943218946457, -0.23675239086151123, 0.6730165481567383, 0.2399207055568695, 0.8491039276123047, 0.5206871628761292, -1.213399052619934, 0.38760221004486084, 0.15080823004245758, -0.9503508806228638, -1.3767390251159668, -2.463559627532959, 0.7402901649475098, 0.4222331941127777, 0.25076428055763245, -0.5673432350158691, -1.0234906673431396, 0.4684523940086365, 0.665741503238678, 0.9728174209594727, -0.8472039103507996, -1.149431824684143, -0.7003487348556519, 0.6237083673477173, -0.29516521096229553, 0.5553119778633118, 0.3185468912124634, 0.9848663806915283, 0.9508870244026184, 0.7490845918655396, 0.79369056224823, -0.15491345524787903, 0.8516709804534912, -1.1629701852798462, 0.40566444396972656, 0.5714194178581238, 0.3446887135505676, 0.6491987109184265, -1.0142757892608643, -1.0660039186477661, 0.06992863863706589, -0.5081144571304321, -2.9595582485198975, -0.8495392799377441, -0.040113985538482666, -0.8525678515434265, -0.56050705909729, 0.007422644644975662, -0.3420223891735077, -0.3668840527534485, -1.2999242544174194, 1.062705397605896, -1.1382807493209839, 0.4569792151451111, -0.8454862236976624, -1.4827998876571655, -0.5264416337013245, -0.046467140316963196, -0.20870742201805115, 0.48549625277519226, 0.7840997576713562, -0.24587726593017578, -2.36686635017395, 1.101228952407837, -0.7439188957214355, 2.1271860599517822, 0.05767340213060379, 1.9042935371398926, 0.9716579914093018, -2.6020190715789795, 1.1734521389007568, -1.2297582626342773, -2.8716623783111572, -0.5527127385139465, 0.34490129351615906, -1.2017146348953247, 0.5980145335197449, -1.0144155025482178, 0.1631813496351242, -2.8451876640319824, -0.591456413269043, -0.07476871460676193, 0.7593768835067749, -1.380698323249817, -0.19605617225170135, -1.0168449878692627, 0.8148908019065857, -1.120477318763733, 0.6123632788658142, -0.6979224681854248, 0.24443666636943817, -0.12623640894889832, 0.5165695548057556, -1.0854814052581787, -0.2964510917663574, 0.7227491140365601, 0.1538149118423462, -2.9504828453063965, -0.8782012462615967, -1.3928300142288208, -0.40041062235832214, 0.9721799492835999, 0.8836673498153687, 0.8116872310638428, -0.001045128796249628, 0.41992467641830444, 0.3095174729824066, 0.03358178958296776, 0.29893985390663147, -0.11357339471578598, 0.813883900642395, 0.17569999396800995, 0.34824153780937195, -1.2134846448898315, -0.7443093657493591, 0.4435953199863434, 0.4392654299736023, 1.053634524345398, 0.7410240769386292, 0.7874281406402588, 0.5184535980224609, -0.3273848593235016, -0.7562139630317688, -0.38312914967536926, 2.1574151515960693, 0.07971206307411194, 0.05848250538110733, 0.3983365297317505, -0.7978482246398926, -0.7214251160621643, -0.20223145186901093, 0.5474686622619629, 0.7868182063102722, -0.7086954712867737, -0.10798188298940659, -1.3375667333602905, 0.8815129995346069, 2.8184213638305664, -0.290804386138916, -0.38580647110939026, -0.8067769408226013, -0.02228616364300251, -0.27378129959106445, 0.5769370794296265, 0.6396388411521912, 1.1363983154296875, -0.3486810624599457, -0.3323444128036499, 0.22974033653736115, -0.7926415205001831, -0.46279171109199524, 0.20327472686767578, -0.12432555854320526, -0.41671040654182434, -0.42434990406036377, -0.6010981202125549, -1.5779155492782593, -0.675398051738739, 0.30768924951553345, 0.7100412249565125, 0.8680428266525269, 0.8600013852119446, 0.039895061403512955, -0.47594285011291504, 1.3569287061691284, 1.0370858907699585, 0.11678396165370941, 0.6045185327529907, 0.9826883673667908, -1.2758582830429077, -0.6070191264152527, -0.32257041335105896, 1.122606873512268, 0.41478684544563293, -0.833266019821167, -1.609390139579773, 0.20135170221328735, -0.7211712598800659, 0.07869815826416016, -1.4883460998535156, -1.93386971950531, 0.14125150442123413, 1.160022497177124, -0.39026564359664917, 0.6897138953208923, 0.0795108899474144, 0.6666702032089233, 1.315385341644287, 0.38340702652931213, -0.8335326910018921, -0.9198347330093384, 1.6940242052078247, -0.6260318756103516, -0.7042043805122375, -0.12598200142383575, 0.2556757628917694, -1.9478516578674316, -0.41996923089027405, 1.2393730878829956, -0.3164916932582855, 0.13559815287590027, -1.495908498764038, -0.4720032215118408, -1.5948961973190308, 0.8956239819526672, 0.2405233234167099, -0.3951362073421478, -0.137520894408226, 0.05595900118350983, -1.6336842775344849, 1.0647369623184204, -0.9444835186004639, 0.6753054261207581, 1.1563652753829956, 0.19331121444702148, 2.020387649536133, -0.31342026591300964, 0.9275603890419006, 0.9889607429504395, 1.006148099899292, 0.17908068001270294, 0.2874959409236908, 1.56124746799469, -1.18265700340271, 0.9462077021598816, -0.6145365834236145, -2.113196611404419, 1.8245880603790283, -0.6597588658332825, -1.3987622261047363, -2.16308856010437, -0.8458036184310913, 0.6224424242973328, -0.4297308027744293, 0.554466962814331, -0.28034543991088867, 0.19820722937583923, -1.522106647491455, 1.407339334487915, -1.786643147468567, -1.0205078125, -0.3658207058906555, 2.7155847549438477, -1.26613450050354, 0.08129842579364777, 0.4175812304019928, 0.7131555080413818, 0.244602233171463, 0.9988598227500916, -0.9232031106948853, -0.13778094947338104, -1.1838371753692627, -0.5656363368034363, 0.4363519549369812, 1.0740935802459717, -1.47913658618927, 0.7042697668075562, 0.6556914448738098, 1.6525330543518066, -2.509014844894409, 1.2570511102676392, 1.4196815490722656, 0.9328754544258118, -1.1371617317199707, -1.6185933351516724, 1.2423207759857178, -0.6012492179870605, 1.5429422855377197, 2.1145949363708496, -0.3076932430267334, -0.08449006080627441, 0.2427569180727005, 1.5141521692276, -0.28890013694763184, 0.2583470046520233, -0.5654760599136353, -0.0717565193772316, -1.2764583826065063, -0.3473718464374542, -0.3687518835067749, 1.2438039779663086, -0.998130738735199, -0.34091371297836304, -0.05531821399927139, 1.7673553228378296, 1.1304525136947632, 0.1311681568622589, 1.6945797204971313, -1.1990338563919067, 1.2811635732650757, -1.2925522327423096, -0.31111589074134827, 0.6245649456977844, 0.10875678807497025, -0.5756350159645081, 1.4509791135787964, 1.23008131980896, -0.6143832802772522, -0.9802819490432739, -0.11859074234962463, -0.7053857445716858, -0.8941380381584167, -1.174578309059143, -1.7642332315444946, 1.1057003736495972, 1.6461193561553955, -0.7838671207427979, -0.0665324404835701, -0.9656898975372314, 1.1430411338806152, 0.03412996232509613, 1.2590049505233765, -0.8623884916305542, 1.5628896951675415, 0.15350866317749023, -1.0054141283035278, 2.365532159805298, -0.015000567771494389, 1.1920909881591797, 0.975132405757904, 1.6539766788482666, 0.6648771166801453, -1.2348304986953735, 0.5147007703781128, -0.6819843053817749, 0.08139991015195847, 1.21766197681427, -0.979858934879303, -0.6338540315628052, 1.0744571685791016, 0.4754084646701813, 0.14736570417881012, 0.19222798943519592, -0.07052832841873169, -1.5849661827087402, -0.12705744802951813, -0.8414870500564575, -0.16690078377723694, 0.3628009855747223, -0.41243255138397217, -1.0343348979949951, 0.36318737268447876, 0.2746967077255249, 1.8087353706359863, -0.8915607333183289, -0.6896329522132874, 2.407696485519409, -0.8918479084968567, 0.7057222723960876, -0.6964902281761169, 1.4660429954528809, -1.6035919189453125, -0.6961804628372192, 1.5120023488998413, -0.8996055126190186, -1.022038459777832, -0.5432865023612976, -1.7425920963287354, 0.37810376286506653, -0.9373833537101746, -0.21971996128559113, -2.0381791591644287, -0.11413482576608658, -0.9188380241394043, -1.0918653011322021, 4.035831451416016, 0.2691517770290375, -1.038590431213379, -0.20987586677074432, 0.7165651321411133, 0.6760463714599609, -0.7323750853538513, -0.6396273970603943, 0.08140233159065247, 1.3288466930389404, -0.6963879466056824, 0.8848288655281067, 1.0692428350448608, 0.6846866011619568, 0.7414034605026245, -0.2518633306026459, 0.6327313780784607, -0.8420277833938599, -0.21753746271133423, 1.0069119930267334, -0.3605363070964813, -0.31001976132392883, 0.5093071460723877, -0.005240545142441988, 0.132786825299263, 0.35360440611839294, -0.4354967474937439, -1.735487461090088, -1.6210514307022095, 2.36403751373291, 0.11234284937381744, 0.7071751356124878, 0.6479505300521851, 0.1655558943748474, -0.3562960624694824, -0.014144698157906532, -0.4132702648639679, 0.5606969594955444, -1.163265585899353, -0.49946269392967224, -1.1640211343765259, -0.7439384460449219, 0.14496737718582153, 1.0483986139297485, 0.018671423196792603, -0.3463667333126068, -0.3554379642009735, 0.2306281477212906, 1.3460923433303833, 1.0619823932647705, -0.11443396657705307, -0.7599983215332031, -0.8543879389762878, -0.5914198756217957, 0.8185623288154602, -0.5628668665885925, 0.2566641867160797, 1.3414148092269897, 0.728271484375, 2.159231185913086, 1.88454008102417, -0.5712294578552246, -0.5096755027770996, 0.7601903080940247, 0.45917409658432007, 0.3823302686214447, -0.3569320738315582, -1.0917387008666992, -0.18566255271434784, 1.2597724199295044, 1.3594430685043335, -0.05631810054183006, 2.6649086475372314, 0.31847548484802246, 0.6617667078971863, 0.3474264442920685, 1.6525249481201172, -1.9547746181488037, 0.20779389142990112, -1.0802969932556152, -1.0951594114303589, 1.3828271627426147, 0.41153421998023987, -1.9559730291366577, -0.12537674605846405, -2.0644633769989014, 0.11577367782592773, -3.272841453552246, 0.38300272822380066, 0.7294732332229614, -2.1840195655822754, 0.7933679819107056, 0.11983119696378708, 1.8261725902557373, 0.6619197726249695, 1.0890766382217407, 0.1675315648317337, 2.410282611846924, -0.37752288579940796, -0.04695563390851021, 2.165579080581665, -0.841092050075531, 0.15827184915542603, 0.23240160942077637, -0.2652757167816162, 0.21946053206920624, -0.1900673359632492, 1.368325114250183, -0.8545222878456116, 1.3963072299957275, 0.635721743106842, 0.5115038156509399, 2.1043601036071777, -1.0089789628982544, 0.31980547308921814, 0.36958760023117065, 0.01460594404488802, 0.9885645508766174, 0.29000064730644226, -0.9307242631912231, -1.1121833324432373, -0.9006462097167969, 0.07020930200815201, 1.8588311672210693, 0.15551824867725372, -0.07861378788948059, -1.2023398876190186, -1.2670623064041138, -0.6318466067314148, -0.12229935079813004, 0.6904721260070801, 0.3664245903491974, -1.6604818105697632, 1.122093677520752, -0.19072666764259338, -1.3444889783859253, 0.4216165244579315, -1.4194878339767456, -1.427021861076355, -0.9205241203308105, -0.08303650468587875, 0.38453051447868347, 0.47753480076789856, -0.6305606961250305, -0.013620467856526375, 0.14274150133132935, -1.7534373998641968, 0.4946276843547821, 0.5909740924835205, 0.09734651446342468, 0.7316433191299438, -0.3506143093109131, -0.42222070693969727, 0.6749715805053711, 1.4369659423828125, 0.09827204793691635, -0.34281831979751587, 0.49296101927757263, 0.4119218587875366, -1.9433040618896484, -0.09511946141719818, -0.347079873085022, -2.508445978164673, 0.2736974358558655, 2.605299711227417, -1.602935791015625, -1.3619407415390015, -1.4243773221969604, -0.04722090810537338, 0.585938036441803, -0.617493212223053, 0.27778565883636475, 0.25607243180274963, 0.3254828155040741, 0.3093354403972626, -0.2767372131347656, -0.4152570962905884, -0.7653185725212097, 1.830720067024231, -0.33710992336273193, -0.7232856750488281, 0.07036133855581284, -0.7391515970230103, 1.4077634811401367, -0.976567804813385, 0.9240097999572754, -0.182964488863945, -0.6093250513076782, 0.6096699237823486, 1.2081433534622192, 0.32927843928337097, -1.5948505401611328, -0.5812814831733704, 1.2124255895614624, 0.40798360109329224, 0.8297663331031799, -1.663631796836853, 0.9846630096435547, 0.23399896919727325, -0.08854156732559204, 0.4616093635559082, 0.23966029286384583, -0.024456631392240524, 0.5113065838813782, 0.5949205160140991, -0.65095055103302, -0.038884203881025314, -2.0339741706848145, -0.2110036164522171, -0.6074014902114868, 0.5291929244995117, 0.9117470979690552, -0.5201307535171509, -1.0486358404159546, -1.1972910165786743, 0.1366870254278183, 0.5258899927139282, 1.0161665678024292, 1.5945290327072144, 0.29566797614097595, 1.2836956977844238, -0.47073793411254883, -1.104055643081665, 0.694808840751648, -0.31051191687583923, 2.230570077896118, -0.10277816653251648, -0.41476717591285706, 1.324795126914978, 1.1391661167144775, 0.24189147353172302, -0.01491077896207571, 1.2871489524841309, -0.46603238582611084, 0.45540934801101685, -0.01954367384314537, -0.21840964257717133, 2.4833998680114746, 0.2745780646800995, 0.24429020285606384, 0.5084102749824524, 0.03923293948173523, -0.22855940461158752, -0.4324864149093628, -1.4691060781478882, 0.22657674551010132, 1.4874417781829834, -0.9045449495315552, 1.2624481916427612, -0.3240775763988495, 1.081852912902832, -0.43508800864219666, 1.2635161876678467, 0.028398359194397926, 0.003658383386209607, 2.6603550910949707, -1.0424541234970093, 0.20153307914733887, -1.4858102798461914, 0.7349552512168884, 1.2934603691101074, -1.2532615661621094, -1.1481915712356567, -1.7668161392211914, -0.6973883509635925, 0.3969218134880066, 0.20911194384098053, 0.7591487765312195, -0.9971799850463867, 0.34309566020965576, 1.6976728439331055, -1.76353919506073, -0.2170182466506958, -1.346569538116455, 0.3299192786216736, 1.2630149126052856, -0.6695626974105835, -1.1654695272445679, 0.2462931126356125, 0.5141832828521729, 1.5680052042007446, -0.013609199784696102, 0.5895012617111206, 0.4781065583229065, -0.5859487652778625, 2.210886001586914, 2.0658395290374756, -0.29281407594680786, 0.3152621388435364, -0.42953699827194214, 0.05723167210817337, 0.8199893832206726, 0.3439600169658661, -0.14406006038188934, -0.8960796594619751, 0.10204576700925827, -0.8179003596305847, -0.4983389377593994, 0.6810212135314941, -1.9800373315811157, 0.05404466763138771, 1.1995768547058105, 0.009224442765116692, -0.10235551744699478, 0.8117226958274841, 1.057765007019043, 0.49578607082366943, 0.9212585687637329, 0.5678992867469788, 1.5082955360412598, -1.1390316486358643, 0.1814560443162918, -0.9911030530929565, -1.2093335390090942, -1.054924726486206, -0.6024760603904724, 0.6948244571685791, -1.4729535579681396, -0.11360646039247513, 0.24932806193828583, 0.2357395589351654, 0.12039123475551605, -1.0462439060211182, 0.612389326095581, -2.1641452312469482, -0.34216782450675964, -0.7678463459014893, -0.5745882987976074, 0.15309201180934906, 0.209300234913826, 1.632032036781311, 0.3031916916370392, 0.34421777725219727, 1.6388568878173828, 1.59917151927948, -0.19756349921226501, 0.9974539875984192, -0.23311510682106018, 1.235054850578308, 0.02129685878753662, -1.0840390920639038, 0.226069837808609, 0.17761047184467316, 0.6500988602638245, -0.9179093837738037, 0.3058675229549408, 0.785610020160675, -0.016604173928499222, -0.4219237267971039, -0.1598215103149414, 1.1330255270004272, 0.4020385444164276, 0.12346172332763672, 0.23204460740089417, -0.8827293515205383, 0.3176281750202179, -0.7453233599662781, 0.36115455627441406, -0.8650150299072266, 1.0372470617294312, -1.1332005262374878, 0.2921384572982788, -1.440079927444458, -0.725460410118103, -0.4731839597225189, -0.7522009611129761, -0.1821356862783432, -0.5438528656959534, 0.10466693341732025, 1.4037964344024658, 0.549346923828125, 1.168375849723816, 1.3692597150802612, -0.875498354434967, -0.6042589545249939, -0.6754313707351685, -1.2451189756393433, -1.6923495531082153, 0.29780372977256775, 0.5252057909965515, -1.8074349164962769, -0.09174341708421707, 0.6132491230964661, 0.9239752888679504, 0.15480239689350128, 1.7576181888580322, 1.1456727981567383, -0.5117028951644897, 0.42720934748649597, 0.4638255834579468, -2.0650219917297363, 0.6712082624435425, 0.17044436931610107, 0.9936842322349548, -0.2875484228134155, 1.0904064178466797, -1.3051031827926636, -0.381352037191391, 0.6931746602058411, 0.9473827481269836, -0.3983227014541626, -0.8801678419113159, -0.2967175245285034, 1.890435814857483, -0.5482332706451416, 0.23563317954540253, -0.5751134753227234, -0.08892767131328583, 1.7063477039337158, 0.9238173365592957, 1.137809157371521, -0.7148817181587219, 0.6817100048065186, 0.30074194073677063, 0.521822452545166, -0.9089031219482422, 1.9199376106262207, 0.5120755434036255, 0.07190242409706116, 0.1868690848350525, -0.10893959552049637, -1.1192055940628052, 1.0290461778640747, -2.092151165008545, -1.6748502254486084, 0.2734028398990631, 0.28783416748046875, 0.8620113730430603, 0.3724055290222168, -1.871110439300537, 0.7356134057044983, 1.3944592475891113, 0.9432687163352966, -1.0797293186187744, -0.5112038254737854, 1.6935648918151855, -0.8924883008003235, 1.2674362659454346, -0.35617387294769287, 0.0040876418352127075, -1.717890977859497, 0.580187976360321, -0.12491054087877274, -3.4875926971435547, -0.2511102259159088, 1.1361743211746216, -1.1632990837097168, -1.3796685934066772, 0.3904587924480438, 0.5215915441513062, 0.7295974493026733, 0.8669453859329224, -0.49081549048423767, 0.682404100894928, 0.5753105282783508, 0.7848463654518127, -0.49396756291389465, -0.206308975815773, -0.39457428455352783, -1.0637670755386353, 1.7664467096328735, -1.7293022871017456, 2.0676188468933105, -0.8118419647216797, 1.934926986694336, 0.08943852037191391, 0.04229452833533287, -0.5168604850769043, -0.6335580945014954, 2.108900785446167, -1.2009841203689575, 0.8185783624649048, -0.3661141097545624, -1.0890496969223022, -1.304132103919983, -0.2575455904006958, 0.005336319096386433, 0.4185599982738495, 1.3385974168777466, -0.12906865775585175, 0.7363258600234985, -0.18443214893341064, -2.0780341625213623, 1.3832563161849976, -0.9677383899688721, -0.044742193073034286, 0.36781662702560425, 0.20252317190170288, 0.7544611692428589, -0.20643948018550873, 0.4985564947128296, -1.3144327402114868, -0.056255921721458435, -0.2984844148159027, -0.32143980264663696, -0.8913424015045166, 0.8218116164207458, -0.19731634855270386, 0.9024612307548523, 0.2754940390586853, -0.19784784317016602, -0.12743671238422394, -0.43656209111213684, 0.5409227609634399, -0.17124012112617493, -0.7424355745315552, 0.7252650260925293, -0.8991382122039795, 2.845202922821045, -0.8456012606620789, -0.05903072655200958, 0.9013837575912476, -1.3762181997299194, -1.6772998571395874, -0.26492804288864136], [-0.3323395848274231, 1.5310598611831665, 0.22008147835731506, 0.7950271964073181, 0.03496205434203148, 0.23895515501499176, -1.4899606704711914, -1.289576768875122, 0.9388719201087952, 0.5088289380073547, -0.2383677065372467, 2.25833797454834, -0.4625014364719391, 1.0976828336715698, -0.4134320914745331, 0.0027838186360895634, -0.9137881398200989, -1.640408992767334, 0.02554217539727688, 1.7272120714187622, 0.8295411467552185, -1.1357784271240234, 0.41695624589920044, 0.3621377944946289, 1.7157127857208252, -0.3056686222553253, 0.20310527086257935, 1.0445847511291504, 0.0016570341540500522, 0.16440774500370026, -1.2701448202133179, 0.09829562902450562, 0.0350542888045311, -0.534462034702301, -0.08952290564775467, 0.6295568943023682, -0.607754111289978, -1.8296458721160889, 1.6117238998413086, 0.7622492909431458, 1.1510905027389526, 0.4222540557384491, 1.2323981523513794, -0.7110381126403809, 0.8634763360023499, 0.23446179926395416, 0.4323456287384033, -1.0021840333938599, 0.505241870880127, -1.3411529064178467, 0.07034952938556671, 0.6622322201728821, -0.13324430584907532, 0.37164896726608276, 0.24819503724575043, 0.6823489665985107, -0.14379265904426575, 0.6261816024780273, -0.07418398559093475, -0.661624014377594, 0.3486268222332001, 1.4903643131256104, -0.30959489941596985, -0.5176772475242615, -0.4128369390964508, 0.7007177472114563, 1.6422410011291504, 0.9551126956939697, 1.5515987873077393, -0.29574963450431824, -0.6971803307533264, 1.867205262184143, 0.5878750085830688, -1.7878170013427734, -1.0167335271835327, -0.2434743344783783, -0.22878655791282654, -0.2952471673488617, 0.23555047810077667, -0.09682035446166992, -0.6665863394737244, -0.46742427349090576, -0.62908935546875, -0.3888447880744934, 1.2384657859802246, -1.2935134172439575, -0.34504351019859314, 1.07843816280365, -0.05881897360086441, -0.900196373462677, 0.11552543193101883, -0.7435939311981201, -1.463732123374939, 0.8147505521774292, -0.84970623254776, 0.39026352763175964, -0.9474228620529175, -1.2365278005599976, 0.2266150414943695, 0.2036724090576172, 0.6912281513214111, 0.7481141686439514, 1.128760576248169, -1.3617802858352661, -0.27292266488075256, 1.3890191316604614, -0.4660308361053467, -2.2243659496307373, -0.9128570556640625, -0.12591944634914398, -0.6915386319160461, 0.12405884265899658, -0.5551593899726868, 0.7984344959259033, 3.5406458377838135, -0.22128865122795105, 0.41678109765052795, -0.8526188135147095, -0.045133814215660095, 0.26542580127716064, -0.46349358558654785, 0.8617926239967346, 0.14913113415241241, -1.209202527999878, 2.497272491455078, 0.7397623062133789, -0.7540364265441895, 0.8929440379142761, 0.10883783549070358, -0.14742633700370789, 0.6092147827148438, -0.7481610774993896, 1.6557785272598267, -1.4931565523147583, -0.3698372542858124, 0.01935122348368168, 1.5330579280853271, 0.4120261073112488, 0.6645538806915283, 0.39257553219795227, 1.222467064857483, 0.13267900049686432, 0.1271335780620575, -1.1012537479400635, 0.14152632653713226, -0.6077423095703125, 0.36652928590774536, -1.8042361736297607, 0.11873624473810196, 3.1185643672943115, 0.987213134765625, 0.2267124056816101, -0.01696009375154972, 2.181713342666626, 0.9342055916786194, -0.04570940136909485, -2.3398735523223877, 0.8948894143104553, -2.0564661026000977, 0.45758572220802307, 0.023629726842045784, -1.355374813079834, 0.9852107167243958, -0.21668939292430878, -0.6299649477005005, 0.38578855991363525, 1.0512961149215698, -0.7539119720458984, 0.2153521180152893, 1.7329620122909546, -0.5751736164093018, -1.4072015285491943, 0.3038289546966553, 0.7398253083229065, -0.19360318779945374, -0.33760198950767517, 1.1645565032958984, 0.4180629253387451, -1.181148648262024, -1.331656813621521, 1.2880157232284546, -0.061398543417453766, -0.8305355310440063, -0.3344184458255768, 0.15754476189613342, 2.7421584129333496, 0.0034780725836753845, 0.5661341547966003, -0.5636476278305054, -1.9745653867721558, -0.3382945656776428, 1.117920994758606, -0.7888727188110352, -2.5658490657806396, 0.002960545476526022, -1.4102635383605957, 1.1012455224990845, -1.0731834173202515, 1.007901668548584, 0.09335055947303772, 0.7910640239715576, 0.19943061470985413, -0.34764364361763, -0.5781893730163574, 1.0592052936553955, -0.9068763256072998, -1.745692253112793, -0.4815449118614197, -2.2490158081054688, -0.5122618675231934, -0.26330751180648804, 0.42111533880233765, 1.7849644422531128, 0.0762510746717453, 1.0378432273864746, -0.03518842160701752, -0.28284138441085815, 0.6860863566398621, -0.629754364490509, 0.656053900718689, -1.3644680976867676, 0.6919856071472168, -1.0500926971435547, 1.3525830507278442, 0.3663387894630432, -0.17141439020633698, -0.7926225662231445, 0.5411360859870911, -0.982532262802124, -0.6692183017730713, 0.7785872220993042, -0.6617321968078613, -0.46821948885917664, -0.6025215983390808, -1.873651146888733, 0.8149431347846985, 0.11415114253759384, 1.612503170967102, 0.578346848487854, -0.6469046473503113, -0.729066789150238, -1.0287632942199707, -1.7280226945877075, -1.2519195079803467, -0.3627561926841736, 0.3209702968597412, -0.7850194573402405, 0.39898809790611267, 0.4114096164703369, 0.7127475738525391, 0.10208144783973694, 0.023709343746304512, -0.2467493861913681, -0.8720407485961914, -0.7809282541275024, 1.48570716381073, 0.3983125388622284, -0.6432180404663086, 1.5800683498382568, -1.0670173168182373, -0.12211470305919647, -0.6515598297119141, -0.24467530846595764, 0.507408618927002, 1.0443129539489746, -0.40294498205184937, -0.4042544662952423, -0.5662808418273926, -1.156493902206421, 0.4717290997505188, 2.0434563159942627, -0.7176265716552734, -1.7149220705032349, -1.4230173826217651, -0.024503624066710472, -0.923583984375, 0.34454649686813354, 1.8023813962936401, -1.4327040910720825, 1.8953347206115723, -1.7144622802734375, 0.21573206782341003, 1.0550707578659058, 0.051397956907749176, -0.8054206371307373, 2.254119396209717, 0.6319282650947571, 0.6715011596679688, -1.6742452383041382, -0.652465283870697, -1.6996036767959595, -0.4025706648826599, 0.28285133838653564, -0.6304325461387634, 0.9486066699028015, -0.2943975627422333, -1.6028999090194702, 0.7372650504112244, 1.592185616493225, 1.8093165159225464, 1.675142765045166, 0.6088255643844604, -1.5202600955963135, -0.5186846852302551, 1.6245025396347046, -1.0204826593399048, -0.040887657552957535, -1.843516230583191, -1.1139166355133057, -1.0165799856185913, -0.33700039982795715, 0.29362553358078003, 0.25775977969169617, -1.9053924083709717, -0.7312320470809937, -0.3180839419364929, -1.122823715209961, 0.4677307903766632, 0.0809856727719307, -1.0349241495132446, -0.687981903553009, -0.5610488057136536, -0.9375326037406921, 0.4717443287372589, 0.9393323063850403, -0.686909556388855, -0.6153789758682251, 0.5985655784606934, -0.02954823151230812, -1.895338535308838, -0.2813202440738678, 0.15644225478172302, 0.7357651591300964, -1.0474966764450073, -0.5505858063697815, 1.614418625831604, -0.1951410472393036, 0.5541836619377136, 0.12470676004886627, -1.4371684789657593, 1.002606749534607, 0.6772042512893677, 3.152357816696167, -0.6818738579750061, 0.5195848941802979, 3.074708938598633, 1.4604527950286865, -0.5291878581047058, -0.062355998903512955, 1.8076671361923218, -0.00433488842099905, 0.5410568118095398, 0.4550834000110626, 1.1762458086013794, -0.9736987948417664, 0.2153230607509613, 0.7925138473510742, -0.469772070646286, 0.4790022075176239, -0.3847156763076782, -1.7493964433670044, 0.41763871908187866, -0.5680859088897705, 0.03978622332215309, 0.4976480007171631, -0.16551436483860016, -0.1745397299528122, -0.3011300563812256, 2.283200740814209, -0.9938299059867859, 0.3913134038448334, -0.6757515668869019, 0.006866730283945799, 1.1749706268310547, 1.3420472145080566, -1.0441006422042847, 1.738742470741272, -1.1490797996520996, 0.49685147404670715, -0.8021296858787537, 0.6278555989265442, -2.940288543701172, -0.41910770535469055, 1.5304486751556396, 1.9029903411865234, 1.4432346820831299, 0.8119653463363647, -0.44586288928985596, -2.1862990856170654, 0.07561344653367996, -1.1322283744812012, -0.9931143522262573, 1.1670024394989014, 1.0059159994125366, 0.5942980647087097, -1.2051769495010376, 0.504119336605072, -0.2632535994052887, -0.2680959403514862, 0.7849671840667725, -0.049597516655921936, -1.8124369382858276, -0.015099086798727512, -0.4056815803050995, 1.1142433881759644, 0.45293760299682617, -0.32662975788116455, -0.3974958062171936, -0.1270325779914856, 0.3695201575756073, -0.9692707657814026, -0.4394434690475464, -0.48769810795783997, 0.3208885192871094, -0.25812283158302307, -1.398602843284607, 0.5369000434875488, 0.3331185281276703, -0.3384828269481659, -0.2297389656305313, -0.5811625123023987, 0.687741756439209, -0.18183279037475586, -0.5718918442726135, -0.6549094319343567, 1.9066166877746582, -0.337367981672287, 0.15444514155387878, 0.5461618304252625, 1.6084367036819458, 0.7323858737945557, 0.02646566741168499, -1.4279334545135498, -0.740273118019104, 1.3448208570480347, 0.8241590857505798, -1.2401938438415527, -1.3482333421707153, -1.5438674688339233, 0.6766535639762878, -1.0694786310195923, 0.5800732970237732, 1.0555177927017212, -1.2095565795898438, -0.4543180763721466, 1.6601234674453735, 0.6357041597366333, -1.0895682573318481, 0.6580765247344971, -0.0787549838423729, -0.07844642549753189, -0.316605806350708, -1.1823244094848633, -1.6438740491867065, 1.19237220287323, -0.9187211990356445, -1.4515351057052612, 0.6704646944999695, 0.5440508127212524, -0.8324267864227295, -0.8069587349891663, 1.6011706590652466, -0.22436140477657318, 0.18207238614559174, 0.44287726283073425, 0.7869285941123962, 0.8968422412872314, 0.09270859509706497, -0.6901427507400513, -0.3562891483306885, 0.061673618853092194, -1.0959367752075195, -0.565309464931488, -0.17253707349300385, 0.2349046915769577, -1.3455379009246826, 0.8358003497123718, 0.039235230535268784, -1.2427932024002075, -0.6040310263633728, -0.7396649122238159, 0.6403497457504272, 0.395927757024765, -0.21338587999343872, -1.0017935037612915, -1.9676512479782104, -1.6779394149780273, 0.8158235549926758, -0.5446043014526367, 1.2683464288711548, 0.043361686170101166, -0.40960636734962463, -1.08468759059906, 1.3746047019958496, 0.43634670972824097, -1.3604069948196411, 0.025981655344367027, -1.6268548965454102, -0.2537194788455963, 0.37647536396980286, 1.4013993740081787, 0.49093183875083923, 0.35533851385116577, 1.3840596675872803, -1.200373888015747, 1.0143171548843384, -1.2996083498001099, -1.4567246437072754, -2.411431074142456, -0.3740493655204773, 0.8327821493148804, -1.849522352218628, 0.30189308524131775, 2.0258753299713135, 0.5959957838058472, 2.055731773376465, 1.6671864986419678, -0.7090949416160583, 1.8649253845214844, 0.3449322283267975, 0.5141693353652954, 0.7122786641120911, 0.13932755589485168, -0.18447177112102509, 1.1694810390472412, 0.08107013255357742, -1.443350911140442, -0.9904373288154602, 0.8367828130722046, 1.0309010744094849, 2.0966989994049072, -0.5760770440101624, -0.3324955403804779, -0.0440288707613945, -0.2764036953449249, -3.223374128341675, -0.0888550654053688, 0.01921677030622959, -1.0614233016967773, 0.4918733537197113, 0.0002358633792027831, 0.7019610404968262, -0.40632784366607666, -0.3703126013278961, 0.7933993339538574, -0.31451019644737244, -1.4153896570205688, 0.6091157793998718, -0.35340291261672974, 0.2744956314563751, -1.137831211090088, -0.22237807512283325, -0.42043372988700867, -0.09853418916463852, -0.7175753116607666, -0.19577659666538239, 0.6188201904296875, -1.884247064590454, 1.0317378044128418, -0.684577465057373, -0.927682638168335, -1.4343301057815552, -0.9826425313949585, -0.3649371266365051, 0.9984397292137146, 0.1468726098537445, -0.21943967044353485, -0.3502762019634247, -0.5842832922935486, -0.6754908561706543, -0.14517994225025177, -1.6679054498672485, 0.8723537921905518, -0.7288311719894409, -0.318677693605423, -1.0849004983901978, 0.15196385979652405, 0.9259187579154968, 0.5846156477928162, 0.302341490983963, -1.097790002822876, -0.2856106460094452, -0.7257355451583862, 1.6108042001724243, 0.06689707934856415, 1.130142092704773, -2.1367599964141846, 1.4218738079071045, 0.5866188406944275, -1.4192429780960083, -0.5213876962661743, -0.6409752368927002, -0.40638288855552673, -0.34447064995765686, 0.13849854469299316, -0.73209547996521, 0.4807351529598236, 0.582473874092102, 0.4333862066268921, -0.9875528216362, -0.09541005641222, -1.836962103843689, 0.32017630338668823, 0.3573834002017975, -1.3010989427566528, 2.1842660903930664, -0.562615692615509, -1.8336167335510254, 1.8349031209945679, 0.11402945220470428, -2.268970489501953, 0.7847274541854858, 1.5638328790664673, -0.016186490654945374, 2.1326656341552734, 0.29737889766693115, 2.2280192375183105, 1.2669658660888672, 0.01968645490705967, -1.0097851753234863, -0.8865178823471069, 0.990948498249054, -1.191752314567566, -1.262369990348816, 0.8970928192138672, -0.4560787081718445, -1.037430763244629, 0.1453443467617035, -0.9485608339309692, 1.018941044807434, 0.836648166179657, -1.9460724592208862, -0.04418642818927765, -0.16976696252822876, 1.203883171081543, 0.0559149831533432, 0.585245668888092, -0.3720453083515167, -0.39531412720680237, -0.4626087546348572, 0.4915183484554291, 0.18290133774280548, -0.7401914596557617, -0.7357001304626465, -0.6118977665901184, 0.48689204454421997, -0.14767463505268097, 0.6042243838310242, -1.7147245407104492, -0.5535972714424133, -0.8206824660301208, 0.20599690079689026, -0.11252181231975555, -0.15862415730953217, 0.3926774263381958, -0.6988459229469299, 2.68729567527771, -0.7904598116874695, 1.4868934154510498, -0.4448930621147156, 1.2352944612503052, 1.0195952653884888, 0.30883413553237915, -0.5613095760345459, -0.23336929082870483, -0.29176604747772217, -1.9300835132598877, 3.468702793121338, -0.6810787916183472, 0.6385719776153564, -2.0063087940216064, 0.4469040036201477, 0.13468295335769653, 0.807723343372345, 0.535190224647522, -1.1043710708618164, -0.2700812816619873, 0.5270364284515381, -0.6086351871490479, 0.9738329648971558, -0.45383375883102417, -0.0520632229745388, 0.6124048829078674, -0.7880520224571228, -0.9069091081619263, 0.8311938643455505, -1.3359020948410034, 1.1883368492126465, 0.07331842929124832, 1.5430572032928467, -0.9329763054847717, 0.258329838514328, -1.1565520763397217, 1.3267918825149536, -0.17253781855106354, -0.6325662136077881, -0.7626389861106873, 1.7903255224227905, 0.6496967673301697, -0.22759197652339935, 0.8728817105293274, -0.11487038433551788, 0.33775728940963745, 0.04258762300014496, 0.5703560709953308, -1.475070834159851, -0.44687768816947937, -0.5771598815917969, 0.7528823614120483, -0.7412038445472717, -1.026228427886963, 0.11670161038637161, -1.4924198389053345, -0.16282027959823608, -0.6091793775558472, -0.8200855255126953, 1.7340750694274902, -0.2136422097682953, 0.773338794708252, 0.5802711248397827, 0.064166359603405, -0.7098802328109741, -0.23909816145896912, -1.4210803508758545, 1.340936303138733, -0.2899973392486572, 0.358940064907074, -1.5788244009017944, 1.0773017406463623, -0.18584437668323517, -0.5437451004981995, 1.6154966354370117, 0.06815656274557114, -0.06073711812496185, 0.09515000134706497, 0.12145008891820908, -0.6788951754570007, 0.3042204976081848, 0.6916176676750183, -0.8020411729812622, 1.2163900136947632, 0.6036083698272705, 0.49091365933418274, 1.672985553741455, 0.5743329524993896, -0.5601297616958618, 1.0314851999282837, -0.052947621792554855, 0.5594096779823303, -2.3967337608337402, 2.0733439922332764, 1.530131459236145, 0.04367029294371605, -1.9905009269714355, 0.4920516014099121, -1.4796751737594604, -0.9714235067367554, 0.3389667570590973, 2.14204740524292, -0.7363020181655884, -0.08256883919239044, 0.5741837620735168, 1.6418428421020508, 1.3286534547805786, -1.1414679288864136, -1.6782338619232178, 1.0756685733795166, 1.4997941255569458, -0.4826795160770416, -1.798903465270996, -0.3516741096973419, -0.6245744824409485, -1.3040412664413452, 1.312881588935852, -1.8994094133377075, -1.2966282367706299, -0.7960547804832458, -0.26187190413475037, 0.39834246039390564, 0.2970592975616455, -0.41679975390434265, 1.2326594591140747, -1.2157108783721924, 0.7493404746055603, 1.637568712234497, -0.17957590520381927, 0.8483840823173523, -1.2165910005569458, -0.07108457386493683, 1.4442347288131714, 0.8716850280761719, -1.1174935102462769, -0.1427447497844696, 1.157913088798523, -1.2345771789550781, -0.3840045630931854, 0.4522475302219391, -0.313971608877182, -0.633725643157959, -0.16062632203102112, -0.22723130881786346, 0.5062645077705383, -0.47094157338142395, -0.25992196798324585, 0.42751872539520264, -0.44604694843292236, -0.4464586079120636, -0.7562330961227417, -0.03489207848906517, -1.093990445137024, 2.1930112838745117, -0.22803886234760284, 1.2729711532592773, 0.2572851777076721, -0.7534996271133423, -0.6328127980232239, 0.6522453427314758, 0.11716299504041672, 0.13973268866539001, -0.2448451966047287, -0.12072848528623581, -0.9354032278060913, 0.3669988512992859, 0.7777686715126038, -0.8309670686721802, 0.17117977142333984, 0.24984045326709747, 0.11329653859138489, -0.4661630094051361, -0.014564274810254574, -0.7100714445114136, -1.815983533859253, -0.8793003559112549, 0.8872597217559814, 0.5212962627410889, -0.8889064788818359, 0.7216688990592957, 0.07314442843198776, -1.9091107845306396, 0.8745834231376648, 0.714996874332428, -0.24972382187843323, -1.27029287815094, 1.2089697122573853, 0.9166733026504517, 0.5527569651603699, 0.7830357551574707, 0.7319918274879456, -0.5742198824882507, -0.4395691454410553, -0.06706182658672333, -1.5220787525177002, 0.422054260969162, -1.2202507257461548, 1.757041573524475, 0.8461204171180725, 2.263545513153076, -0.36996594071388245, -0.8502855896949768, 1.1523375511169434, 0.14382681250572205, 1.1365926265716553, -0.16511546075344086, -3.484607696533203, -1.090308666229248, 0.3289373517036438, -0.21682003140449524, -0.29390132427215576, -0.8755784630775452, -0.14342115819454193, 0.26425471901893616, -0.4728999733924866, 0.26392099261283875, -1.138574242591858, -0.5146132707595825, 0.7169309258460999, 1.0362284183502197, -2.3129494190216064, 0.6694947481155396, -0.7864499688148499, 2.336392879486084, -1.121681571006775, -2.0804173946380615, -1.300045371055603, 0.02795718051493168, 0.3251666724681854, -0.13659584522247314, -0.7357780337333679, -1.1317710876464844, 0.27406859397888184, 0.7138531804084778, -0.39398324489593506, -0.9351706504821777, 2.053260326385498, -0.09343110769987106, -0.11631253361701965, 0.9819414019584656, -0.5755100846290588, -1.8901536464691162, -1.2532646656036377, 1.893546462059021, 0.48077118396759033, -2.0021157264709473, -2.3254165649414062, -1.35670804977417, 1.0756020545959473, 0.937812089920044, 0.36340028047561646, -1.0029939413070679, -0.06906667351722717, 1.1047254800796509, 0.9954172968864441, 0.6118156313896179, -1.0937036275863647, 0.178226500749588, -0.08231232315301895, -3.0138587951660156, 0.21178747713565826, -2.5487372875213623, 1.656564474105835, -0.3344946503639221, -1.5964722633361816, -0.36622199416160583, -0.5738103985786438, 0.7446733117103577, 0.0008267433731816709, 0.8749635219573975, -0.20213454961776733, 0.42514723539352417, -0.08585743606090546, 0.7151023149490356, 0.10133058577775955, 0.7699286341667175, -0.6969799995422363, 0.5595439076423645, -0.15455153584480286, 0.959036111831665, -0.5630963444709778, -0.5215591788291931, 0.30456024408340454, 0.6409436464309692, 0.747484028339386, -1.0933129787445068, 0.6770893335342407, -0.5380998253822327, -0.4504014253616333, 0.7401602864265442, 0.23401136696338654, -2.2872955799102783, -1.1965805292129517, 1.0979890823364258, 2.2073729038238525, -1.2114375829696655, -0.11976300925016403, 1.8268135786056519, -0.44511678814888, 0.05560234189033508, 0.8923400044441223, -1.411768913269043, 0.5463669896125793, -0.47741663455963135, 1.5341256856918335, 1.3272228240966797, 1.2472522258758545, -1.0427182912826538, 0.3146316409111023, -1.1844651699066162, -0.6912863254547119, -0.05825268477201462, 0.8074085712432861, -0.3886045813560486, -1.1226727962493896, 0.31030404567718506, -0.12400532513856888, -0.0344218984246254, -0.19024857878684998, 0.18743249773979187, 2.0207977294921875, -0.047895532101392746, -1.891106367111206, -0.4002699553966522, 0.4450512230396271, 0.7820057272911072, 1.490628957748413, -0.6675593256950378, 0.18861104547977448, 1.191016435623169, 1.2331078052520752, 0.13366873562335968, 1.4241026639938354, -2.310877799987793, 1.5717495679855347, 0.9494450688362122, -1.093163013458252, -0.023991266265511513, 0.8850739002227783, -0.7479168772697449, 0.5615534782409668, -1.1090476512908936, 1.4710382223129272], [0.46362534165382385, -0.03750412166118622, 1.3179186582565308, -0.5423476696014404, 1.3895611763000488, -0.4118000566959381, 0.8833246231079102, 0.210345059633255, 1.3898701667785645, 0.8145060539245605, 0.50276118516922, -0.007086527533829212, -0.6491461992263794, -0.3061774671077728, -0.4892645478248596, -0.3515308201313019, 2.159111261367798, -0.1586223691701889, 0.4515068233013153, -0.42670589685440063, -0.7511565685272217, 0.8901064395904541, 1.2903060913085938, -0.6470887660980225, -0.6603256464004517, -0.7623587846755981, -0.3836021423339844, -1.7206387519836426, -0.6835795640945435, -0.12756778299808502, -0.31051936745643616, -0.2741873562335968, 2.6184093952178955, 0.30548933148384094, 1.0011640787124634, -0.31339526176452637, 0.6637770533561707, -1.4835189580917358, 0.42865118384361267, -1.7134828567504883, -0.02439388446509838, 1.2051690816879272, 0.011171362362802029, 1.738318920135498, -0.6806217432022095, -1.2058420181274414, -1.856640338897705, -1.1049283742904663, -1.4905668497085571, -0.8333290815353394, 0.7024468779563904, 0.9461675882339478, 0.9043805599212646, -0.7690191268920898, 1.9553178548812866, -1.07245671749115, 0.7090030312538147, -0.46329614520072937, 0.036721959710121155, 0.7158447504043579, 0.4019712209701538, 1.9775793552398682, -0.4146840572357178, 0.46572521328926086, -3.240149974822998, 1.1276907920837402, 1.201841115951538, 1.9433339834213257, -0.8915683031082153, 0.5770630240440369, 0.503793478012085, 0.7138078808784485, -1.805275321006775, 0.9405368566513062, -0.37776562571525574, 0.3787781298160553, 0.09979778528213501, 0.7079234719276428, -0.9307211637496948, -1.2845267057418823, -1.7087479829788208, 0.34601181745529175, 1.0193512439727783, 1.2864056825637817, -0.3295934200286865, -0.07776284962892532, 2.2809934616088867, -0.38478466868400574, 0.7072234749794006, 1.031609058380127, 0.6100339889526367, -0.04781556501984596, 0.5603771209716797, -1.6227211952209473, -1.1840022802352905, -1.2572839260101318, -0.1758890300989151, -0.9023205637931824, -0.06354807317256927, -0.6001418232917786, -1.4252676963806152, 1.7052216529846191, 1.1644830703735352, 1.3042899370193481, -1.014418363571167, 1.1050307750701904, -0.6753038763999939, -0.7723690867424011, -0.5104920864105225, 0.9429838061332703, -1.0534652471542358, -0.4967985153198242, 0.25187304615974426, 1.1185907125473022, -1.0620548725128174, 0.37680086493492126, 0.33951717615127563, 1.2566401958465576, 0.02860773354768753, 0.47582587599754333, -1.9570797681808472, 1.6396453380584717, -1.5269312858581543, -0.16531409323215485, -0.8985631465911865, 1.1724766492843628, 0.6362006068229675, -2.294278383255005, 0.0781559869647026, -0.9950814843177795, -2.2888762950897217, 0.6848313212394714, 0.1917237490415573, -0.5256348848342896, -1.058464527130127, 0.35466837882995605, 0.8201001286506653, 0.6779192686080933, -0.037608951330184937, 1.4424365758895874, 1.5562515258789062, 0.5312303900718689, 1.137851357460022, 0.9612228870391846, 0.15198728442192078, -1.8934632539749146, -0.10168789327144623, 1.5417826175689697, 1.5929679870605469, 0.3223693370819092, -0.2824936509132385, 0.7303197383880615, -1.0356934070587158, 0.886189877986908, -0.31283020973205566, -0.6478562355041504, 0.6100210547447205, -0.8758144974708557, -1.2002129554748535, 0.6128605008125305, 0.22669042646884918, -0.6084339618682861, -0.10909426212310791, 0.7549681663513184, 1.0681017637252808, -0.292806476354599, 0.553348183631897, 0.9025362133979797, 1.4896248579025269, 0.8810222148895264, 0.979435384273529, 2.109222888946533, -1.0816526412963867, -2.8154938220977783, 1.0495359897613525, 1.27486252784729, 2.2227795124053955, 0.3089020252227783, -2.056098222732544, 0.19663874804973602, 0.17003022134304047, -1.1132352352142334, -0.8447277545928955, 1.279103398323059, 0.7880396246910095, -0.08660732209682465, 0.39167121052742004, 0.09598445892333984, 0.5892450213432312, -1.3771775960922241, 0.26582491397857666, 1.6469148397445679, -0.4652571976184845, 0.5437136292457581, 0.21974903345108032, -1.0163668394088745, 0.4689043164253235, 1.193800926208496, -0.7023134827613831, -0.4200134873390198, 0.9272860884666443, -1.0125455856323242, -2.268561840057373, 1.3106588125228882, -0.31633567810058594, 1.04017972946167, 1.412503719329834, 0.3175102770328522, 0.7710669040679932, 1.5174943208694458, 0.5147969722747803, 1.2617335319519043, 0.358147531747818, -0.2204972505569458, 2.0461199283599854, -0.8631870150566101, -0.06908024102449417, 0.47211170196533203, 0.4050244390964508, -0.7840333580970764, 0.9164914488792419, 0.39674708247184753, 0.3178154528141022, 1.243178129196167, 1.6314361095428467, -1.0810792446136475, 0.07302593439817429, 0.1582810878753662, 2.3510560989379883, 1.0612083673477173, -0.13407550752162933, 0.8219730257987976, 2.3034987449645996, -0.04009626805782318, -0.22646880149841309, -1.0380514860153198, -1.567024827003479, 0.4695296287536621, -0.1432233452796936, 1.553134560585022, 0.45097222924232483, -0.14601382613182068, 0.2518117427825928, 1.0521682500839233, -0.404155969619751, -0.5097215175628662, 0.5036983489990234, -1.379190444946289, -1.2045713663101196, -0.8379083275794983, -1.2647790908813477, -0.1799287348985672, -0.21180200576782227, 0.1389836221933365, 1.004763126373291, 0.6650891304016113, 0.7362032532691956, -0.4250696301460266, -0.4246726632118225, 1.3431986570358276, -0.3129628598690033, -0.06459606438875198, -1.0344902276992798, -2.151262044906616, 0.03055654466152191, -0.14996662735939026, 0.0899737998843193, 1.3922022581100464, 0.141447514295578, 0.11916939914226532, 1.7532821893692017, -0.04759417101740837, -0.285426527261734, 2.1234524250030518, -2.2017147541046143, -0.8552895784378052, 1.0201328992843628, -0.7522121667861938, -0.06687337160110474, -0.889620840549469, 2.0146689414978027, -0.45711439847946167, -0.505702018737793, -1.018308401107788, -1.740451455116272, -2.196859121322632, -0.023869996890425682, -0.4125984311103821, -0.6457679271697998, -1.045403242111206, 1.1491533517837524, -0.531096339225769, 1.5735712051391602, 0.4877541661262512, 0.9293182492256165, 2.216519594192505, -1.039647102355957, -0.2734997570514679, 0.05030454695224762, -0.40191495418548584, 1.2331889867782593, 1.1992275714874268, -0.7575250864028931, -1.231399416923523, 0.968361496925354, -1.1255296468734741, 0.3523770570755005, -2.084388494491577, 0.18261033296585083, -0.7405916452407837, -0.3736530840396881, -2.1789700984954834, 1.8330919742584229, 0.28559088706970215, 0.7747870087623596, -1.3055614233016968, 0.08987091481685638, 0.8689008951187134, -0.5262072682380676, 1.6038737297058105, 0.5325462818145752, 0.4328993558883667, 3.0866589546203613, -0.8060934543609619, 1.3481955528259277, 0.8791254162788391, -0.5585296750068665, -1.6088064908981323, 0.07145361602306366, -0.2711978554725647, -1.5369223356246948, 1.4397366046905518, -0.5691773295402527, -0.8689160943031311, -0.5070216655731201, 0.6807140707969666, 1.0370936393737793, 0.28052961826324463, -0.9522480964660645, -1.403498888015747, 1.8278321027755737, -0.34643808007240295, 0.7951566576957703, -0.18482550978660583, -1.0835261344909668, 0.1581960767507553, 0.09969207644462585, -0.11246418952941895, -0.08862863481044769, -0.5069432258605957, 0.061740465462207794, -0.22270338237285614, 0.026371221989393234, -0.09382583945989609, 0.05422470346093178, -0.5815924406051636, 0.5042039155960083, 0.685845136642456, 2.561640977859497, -0.038264818489551544, -1.711297631263733, -0.5969322919845581, -1.6662135124206543, -0.8850172162055969, 1.040411353111267, -2.0513339042663574, 0.26016589999198914, 0.29082703590393066, 1.186694622039795, -0.03581906110048294, -0.4283982515335083, 1.1171698570251465, 0.13218702375888824, 1.3532822132110596, -0.19518862664699554, -2.418980598449707, 1.6224623918533325, 0.5196574330329895, 0.5994384288787842, 1.7167922258377075, -0.4842463731765747, 0.1773359626531601, -0.47764861583709717, 0.05319463461637497, 0.02584235742688179, -2.132937431335449, -2.301036834716797, -0.718951940536499, -0.6215376257896423, -1.0604928731918335, -0.4741987884044647, 1.3671170473098755, -1.674613118171692, 1.3532432317733765, 0.796040952205658, -0.9437037706375122, 0.049971528351306915, -1.7868068218231201, -0.06968849897384644, -0.1982811540365219, 3.0190539360046387, 0.6032008528709412, -1.0062098503112793, -2.025385856628418, 2.4024975299835205, -0.7808066606521606, -0.0183192677795887, -0.8627350330352783, 0.6345359086990356, -0.4084143340587616, -0.1799054741859436, 0.4436858892440796, -1.2021300792694092, 0.15212774276733398, -1.4124164581298828, 0.6603485941886902, -0.5514931082725525, 0.6286759376525879, 1.3937946557998657, 1.457865595817566, 0.4252966642379761, 0.974753201007843, 0.019805191084742546, -0.11543957889080048, -0.5793472528457642, 0.27686038613319397, -0.23564453423023224, 0.43267855048179626, -0.941130518913269, 1.1106196641921997, -1.4584449529647827, 0.16814294457435608, 0.6648151874542236, 0.23941390216350555, -0.9574108719825745, -0.31191790103912354, 0.7669032216072083, -1.086229681968689, 0.760260283946991, 0.8590556979179382, -0.45210781693458557, 1.1590853929519653, -2.422088384628296, -0.15777470171451569, -0.4954388439655304, -0.40383633971214294, -1.1245757341384888, 0.25477832555770874, 1.415669560432434, -0.0783478170633316, 0.01382124237716198, 0.4637497365474701, -1.291826844215393, 0.74921715259552, -0.6271637082099915, 0.09595068544149399, -0.2945418655872345, -0.5362828373908997, -0.9562223553657532, -0.45492222905158997, -0.12489777058362961, -2.1021504402160645, -0.6487907767295837, -1.9363105297088623, 1.4721759557724, -0.8779392242431641, 0.5754420161247253, 0.6747294664382935, 0.06774527579545975, -0.8261004090309143, -0.09673979878425598, 0.4540157616138458, -0.0541498102247715, 1.1134212017059326, 2.143794059753418, 0.2829238176345825, -0.123375803232193, 1.3282915353775024, 1.8995182514190674, -0.6364913582801819, 0.8681247234344482, -0.3708898723125458, 0.6778170466423035, 0.9117432832717896, 0.29988953471183777, 0.03201039507985115, -0.5828126072883606, 0.8109646439552307, -0.6023384928703308, 0.1450461894273758, 0.5990403294563293, -1.2066928148269653, 0.1599765568971634, 0.19706031680107117, 0.3983514904975891, 0.27422860264778137, -1.87253999710083, 0.024507597088813782, -2.7736856937408447, 0.9785482883453369, -0.2637700140476227, -0.6301537156105042, -1.5695465803146362, 1.6508688926696777, 0.13827426731586456, 0.5468417406082153, -0.2555716931819916, -0.7415622472763062, -0.8066304326057434, 0.9064005017280579, 1.9563966989517212, 0.1871141940355301, 0.5028400421142578, 1.1605404615402222, -0.5537059903144836, 0.7716099619865417, 1.7671953439712524, 0.9582040905952454, 0.028980273753404617, -0.760754406452179, -0.06687450408935547, -1.4323327541351318, -0.6058643460273743, 0.050824590027332306, -0.07376255840063095, -0.521395206451416, -0.6022624969482422, -0.5918259024620056, -0.31197595596313477, -1.9777199029922485, -0.04048316553235054, 0.7719860076904297, -1.8891267776489258, 1.68362295627594, -0.11425638198852539, -0.005974743980914354, -0.41303712129592896, 0.37936362624168396, -0.23292076587677002, 0.9143457412719727, -0.03889007866382599, 0.967833936214447, 1.1044548749923706, -1.1903247833251953, 1.0588479042053223, -1.7076796293258667, 1.83183753490448, -0.09190231561660767, -1.9954549074172974, 2.506476640701294, -1.1061221361160278, 0.7324994206428528, 0.9198176860809326, -0.9099847078323364, 0.3902960419654846, -1.2504292726516724, -1.0382786989212036, 0.16481545567512512, 0.15912315249443054, 0.9384194612503052, -0.8354337215423584, 1.2739710807800293, -1.0576716661453247, 0.32788941264152527, 0.22906918823719025, 0.8511321544647217, -0.9851836562156677, -2.649181365966797, -0.7200575470924377, 0.7217859625816345, -0.7856292128562927, -0.020030643790960312, 1.1903936862945557, -1.3165130615234375, -0.01797148771584034, -0.5160813927650452, -2.048804521560669, -0.3898598551750183, -0.5548214912414551, 2.49735689163208, 0.34088221192359924, -1.58017098903656, 0.09740466624498367, -1.0490024089813232, 0.4448058307170868, 1.1315125226974487, 2.0536744594573975, -1.344259262084961, 0.10061298310756683, 0.8673658967018127, 0.1576881855726242, 0.638464093208313, -0.7077817320823669, 0.04432966560125351, 0.40543535351753235, -1.8774081468582153, 0.6068061590194702, -1.0135504007339478, -0.9547309279441833, -1.9247535467147827, -0.29997944831848145, -0.1938263177871704, -0.7151535153388977, 0.13119062781333923, -1.56742262840271, -0.27846553921699524, -0.3445926606655121, 0.3240598142147064, 0.18063633143901825, 0.16967493295669556, -0.4505169987678528, -1.4779945611953735, -0.5234734416007996, -0.09729795157909393, 0.7168569564819336, -1.1522645950317383, 0.8939367532730103, 0.18856753408908844, 0.4165050685405731, 0.2756560146808624, 0.36681240797042847, -1.842283010482788, 0.36563125252723694, 0.6807245016098022, -0.4536389112472534, -0.6435834169387817, 1.5909836292266846, -0.11571312695741653, -0.4387742877006531, -0.35273799300193787, -0.578502357006073, 0.6377204060554504, -0.7894419431686401, -0.7582025527954102, 0.04106584191322327, 0.173095703125, 0.32053038477897644, 0.9701882004737854, -1.8439998626708984, 0.7015799880027771, -0.7953499555587769, 2.324585437774658, -1.071898341178894, 0.773732602596283, 0.6141272783279419, 0.6896793842315674, -0.430601567029953, -0.009527361020445824, 0.8833725452423096, -1.202195644378662, -1.500770926475525, -0.6586962938308716, -0.3195721507072449, 0.9456457495689392, 0.8260417580604553, -2.0595996379852295, -2.214843273162842, -0.7306226491928101, 0.7987808585166931, 0.647118330001831, 0.4265790283679962, -0.1552101969718933, 0.3081403970718384, 1.413881540298462, -0.5702616572380066, 1.17353355884552, 0.7388697862625122, -1.6355634927749634, 0.4580201208591461, -2.2228708267211914, 1.3189208507537842, -1.1271923780441284, -0.07095661759376526, 0.2378557324409485, 0.462985098361969, 0.9391892552375793, -0.40426987409591675, -0.04348680004477501, 0.05396343022584915, -0.37624427676200867, -0.3133913576602936, 0.32795730233192444, 1.003696322441101, 0.7888562083244324, 1.41997230052948, -0.5898565053939819, 1.3999252319335938, 1.2495019435882568, 0.723214864730835, -0.8488929271697998, 0.9774167537689209, -0.7448388934135437, -0.06296056509017944, -1.0656962394714355, -2.004171371459961, 0.7682884931564331, -0.13813893496990204, 0.09281285852193832, -1.1131576299667358, 0.4428662061691284, 1.8642314672470093, 0.13759267330169678, -0.029574722051620483, 1.0627880096435547, -0.2751464247703552, 1.2938127517700195, -0.7664214372634888, -0.9660771489143372, -0.5830466747283936, 0.2626466751098633, -0.8495487570762634, 0.3694635033607483, 1.3651130199432373, -1.5838693380355835, -0.04745084419846535, -0.44191041588783264, -0.5692890882492065, 0.3847470283508301, -0.16081072390079498, 0.6158222556114197, -0.18592612445354462, -0.6206114888191223, -0.9559960961341858, -0.9804448485374451, -0.7725059986114502, -1.2792341709136963, -1.2250770330429077, -1.2901241779327393, 0.41850942373275757, -0.9431331157684326, 0.13257412612438202, -0.7564085125923157, -0.9682189226150513, -0.6254929900169373, 0.110423743724823, 0.1798459142446518, 0.01944626308977604, 0.6269670128822327, -0.7991662621498108, 0.2183806151151657, 0.6324526071548462, -0.020875783637166023, 2.0928971767425537, -0.8304195404052734, -0.6145360469818115, 0.18222075700759888, 0.8799908757209778, 1.2759721279144287, 0.39317065477371216, 1.270900845527649, -0.25046420097351074, 0.16505494713783264, -2.2923166751861572, -0.4233402907848358, -2.2943594455718994, -1.0651379823684692, -0.5556117296218872, -1.390812635421753, 0.08411748707294464, 0.14970944821834564, 1.3569873571395874, -0.696847140789032, -0.499541312456131, -0.15175390243530273, 0.0347437746822834, 1.4090983867645264, -0.6767818331718445, -1.3913872241973877, -0.4966603219509125, 0.19491824507713318, 0.05687747523188591, 0.47006481885910034, 1.2509387731552124, 0.8446012735366821, 0.37279465794563293, -2.3303658962249756, -0.19690170884132385, 1.3338099718093872, -1.2763715982437134, 0.877768874168396, 1.016496181488037, -1.5879194736480713, -1.2064664363861084, -2.533738851547241, 0.24273627996444702, -0.10353797674179077, 0.6380554437637329, 0.2706388235092163, -0.2232465147972107, -1.4722265005111694, 1.1943475008010864, -0.5858906507492065, 0.6157900094985962, -1.2130557298660278, -0.10593467205762863, 0.31743964552879333, -0.029671812430024147, 0.33969396352767944, 2.0094809532165527, 0.23443089425563812, -1.5540851354599, -0.37120530009269714, -0.93881756067276, 0.27718913555145264, 1.0269744396209717, 0.4206943213939667, -1.613800287246704, -0.840391218662262, 0.8685281276702881, -0.4548566937446594, -0.16367891430854797, 0.9532467722892761, 0.7544958591461182, 2.492997169494629, 0.7717822194099426, 1.3260366916656494, 0.7578064203262329, -0.7147663235664368, -0.7760886549949646, 1.484723687171936, -0.12106048315763474, -0.15408463776111603, 0.7938852906227112, 0.9135398268699646, -0.5984343886375427, -1.695313811302185, -0.05492173880338669, -0.3593311011791229, 0.5686070322990417, 1.5438024997711182, -1.3529597520828247, 0.7009742259979248, 1.3427927494049072, 0.14069034159183502, -0.3601466715335846, 0.17432227730751038, 1.2174023389816284, 0.5446508526802063, 0.17747527360916138, 0.14602385461330414, 0.7135171890258789, -2.4480032920837402, -2.1641318798065186, 0.17959162592887878, -1.040755033493042, 0.14570391178131104, 0.2639251947402954, -0.6127874255180359, -0.3809051811695099, -0.2358195036649704, -1.0038446187973022, -0.32297372817993164, 1.393574833869934, 0.839118480682373, -2.687206268310547, 0.29323288798332214, 0.7096893191337585, 0.19173085689544678, 0.434548944234848, 1.8069484233856201, -0.024534672498703003, -0.6231645345687866, -0.1556403636932373, 0.7150338888168335, 0.9773527979850769, -1.875045657157898, -0.27130985260009766, 0.21038122475147247, 1.519540786743164, 0.10991518944501877, -0.4970262944698334, 1.4267607927322388, 1.692902684211731, -0.7223213911056519, -1.1765903234481812, 0.6992213726043701, -0.32081204652786255, -0.8008060455322266, 0.212628573179245, 1.7234359979629517, 0.1595359444618225, -0.7745447158813477, -0.2542375326156616, 1.3676304817199707, 0.6828928589820862, -0.5401983857154846, -0.42273110151290894, 0.9398238658905029, 0.5649033188819885, 0.2323482483625412, -0.3963472843170166, -0.9399430155754089, 0.3788567781448364, -0.9033420085906982, -0.5232788324356079, -1.900310754776001, 1.6514334678649902, 1.8028088808059692, 1.529680848121643, -0.06273762136697769, -0.06076240912079811, 0.9219450950622559, 1.6540248394012451, -0.06523198634386063, 0.26164913177490234, 1.7237461805343628, 1.6332738399505615, 0.32259076833724976, -0.08173363655805588, 1.3352614641189575, 0.5662206411361694, -0.7120193243026733, 0.5156761407852173, 0.45442694425582886, -1.710606336593628, -0.7149465680122375, 0.2222524881362915, 1.0364365577697754, -0.5311245322227478, -0.8182277679443359, 1.0382959842681885, -0.44159287214279175, -1.0799254179000854, -1.407779335975647, 0.057367898523807526, -0.5263407230377197, -0.1940477192401886, -0.972038745880127, -1.2575843334197998, 1.6498486995697021, -1.3107037544250488, -1.2033215761184692, -0.28535011410713196, 0.1270761936903, 0.7373581528663635, -1.145520567893982, 0.05318373069167137, 0.8236610889434814, -0.4000864624977112, -0.4806475341320038, 0.7108609080314636, 1.4764513969421387, -2.67966628074646, 0.6471316814422607, 1.6255661249160767, 0.2509710490703583, 0.5990725159645081, -0.9317118525505066, -1.4303287267684937, -0.4018516540527344, 1.1215392351150513, -0.019485216587781906, -0.5404415726661682, -1.3399454355239868, 1.9089229106903076, 0.9980443716049194, -0.8301647901535034, 1.3891780376434326, 0.8857485055923462, -0.32202938199043274, 1.678292989730835, 1.1825785636901855, -0.15886875987052917, -1.660957932472229, -1.1736583709716797, 0.3178713321685791, 0.19485138356685638, -0.8676834106445312, 0.7021633982658386, 1.2196767330169678, -0.02512652985751629, -1.6181927919387817, 0.5096431374549866, -1.9623112678527832, -0.3172871470451355, 0.953872561454773, 0.5720204710960388, 0.22957955300807953, -0.5355324745178223, 0.7016210556030273, -0.2997805178165436, 0.16986531019210815, 0.3254718482494354, 0.17096099257469177, 0.5736884474754333, -2.0136847496032715, -1.2641559839248657, -0.17092616856098175, 2.0113229751586914, -0.6635186076164246, -0.16367903351783752, 0.9168208837509155, -0.9952541589736938, 1.3080475330352783, -0.290355920791626, 1.0607154369354248, -0.17735016345977783, -1.8298476934432983, 1.1747931241989136, 0.7526941895484924], [0.8561875820159912, -0.01898304559290409, 0.13693730533123016, 0.3709518611431122, -0.4544490873813629, 0.280487984418869, 0.14776267111301422, 1.4927414655685425, -0.44296279549598694, 1.3314334154129028, 0.1466922163963318, -1.0126475095748901, 1.55300772190094, 0.9108809232711792, -1.2284022569656372, -0.5132860541343689, 1.1536253690719604, 1.2910375595092773, 0.294961541891098, 0.5697349905967712, -0.9551582336425781, 1.8164052963256836, -1.3686871528625488, 0.2678738236427307, 0.25512126088142395, 1.1063847541809082, 0.6331688165664673, -0.3157574236392975, 2.2134010791778564, 0.6464595794677734, -2.3871471881866455, 0.5594468712806702, 0.0314626544713974, 2.013761043548584, 0.7542904019355774, -1.7813829183578491, 0.6374979019165039, 0.5298995971679688, 0.1511329561471939, 0.16972608864307404, -0.16337083280086517, 0.964332640171051, -0.4178518056869507, 1.601519227027893, -1.1323539018630981, 0.7449169158935547, -0.00998042244464159, 0.6784921288490295, 0.15967604517936707, -1.31485116481781, -0.8286014795303345, 1.0782198905944824, 0.8808078169822693, 1.3324110507965088, 1.1028255224227905, 0.44991183280944824, -0.7827917337417603, 0.41413456201553345, 1.6237889528274536, 1.5226935148239136, 0.19852271676063538, 0.5057147145271301, -0.07577189803123474, -0.7548842430114746, -0.6753619313240051, -0.229373499751091, -0.21342045068740845, 2.131263256072998, -0.3696138858795166, 0.8632725477218628, -1.2950541973114014, 1.3639503717422485, -1.4184082746505737, 0.3444826304912567, 0.18236520886421204, 0.9087136387825012, 1.1482484340667725, -0.22326640784740448, 0.1271609365940094, 0.20031116902828217, 1.6356648206710815, 1.3213075399398804, -0.2683373689651489, 1.2578052282333374, -0.6844912171363831, -0.16493681073188782, -1.2347334623336792, -0.0008493026834912598, -0.27735504508018494, -0.6775842308998108, -0.9290348291397095, -0.46424660086631775, 0.3731831908226013, -0.983350932598114, 0.27052149176597595, -1.348049521446228, -1.0167557001113892, -1.4332075119018555, -0.06835772842168808, 1.551891803741455, 1.1185996532440186, -0.7075615525245667, -1.5197303295135498, -0.4591616094112396, 0.04994603991508484, -1.3619921207427979, -0.8095582127571106, -0.5579448342323303, -0.7085379362106323, 0.3470942974090576, -0.7981066703796387, -1.4081674814224243, -3.085740566253662, 0.1081911027431488, -1.036919116973877, 1.3800650835037231, 1.057860255241394, 1.493794560432434, 0.9704517722129822, 1.8512868881225586, 0.6000003814697266, -0.30773597955703735, 0.4251115024089813, 0.6264160871505737, 1.7503725290298462, 0.2902812957763672, 0.36358562111854553, -0.605799674987793, 0.08039466291666031, -0.7527366280555725, 2.9847004413604736, -0.003819685894995928, -0.3006128966808319, 1.8116531372070312, 0.26966550946235657, 2.3856866359710693, -0.2848881185054779, -0.5743746757507324, -0.0874214917421341, -1.1733109951019287, -0.3136948347091675, -2.3784444332122803, -0.9729311466217041, 0.4095768928527832, -0.738342821598053, 3.5785772800445557, 0.8556960821151733, -0.4050738513469696, -1.195787787437439, 1.5038011074066162, -1.1891508102416992, -1.0432616472244263, -0.7527254223823547, -0.0866994559764862, 1.1642072200775146, 0.0644417405128479, 0.3700779378414154, 0.8991064429283142, 1.1163142919540405, -1.7122653722763062, -0.06947395205497742, 0.28254714608192444, -0.17411264777183533, -0.7855830788612366, -0.5168366432189941, 1.407490611076355, -0.5109766125679016, -0.4325459897518158, 1.132372260093689, 0.3835309147834778, 0.41736626625061035, 0.949951171875, -0.056579697877168655, -1.9851187467575073, -0.3255004286766052, -0.15275582671165466, -1.8193341493606567, 0.6985992789268494, 0.7042126655578613, 0.5162679553031921, -1.2078437805175781, -0.03791895881295204, -1.1654365062713623, 1.1317344903945923, 1.0047411918640137, 1.2440563440322876, 0.1355433166027069, -0.7883750796318054, 0.9254956245422363, 0.04787740856409073, 0.47053444385528564, -1.6224966049194336, -0.6097140908241272, 0.5289904475212097, -0.9185715913772583, 0.2232223004102707, 0.6917659640312195, -2.0861988067626953, 1.202233076095581, 0.8138183951377869, 1.3232060670852661, 0.954261064529419, 1.1027811765670776, 2.1363608837127686, 0.8190449476242065, -0.3106701970100403, -1.066375970840454, 0.46682700514793396, 0.6781172752380371, -0.7787641286849976, 0.03173479810357094, -0.5368967652320862, -0.2173856496810913, -0.9015883207321167, -0.17154373228549957, 0.06856326758861542, -0.5863118171691895, 0.265773206949234, 1.3367167711257935, 0.02937231957912445, 0.4746505916118622, -0.8173154592514038, 0.024073781445622444, 0.3696102797985077, -0.7304300665855408, 0.11012861132621765, 0.10183345526456833, -0.704296350479126, 1.2659804821014404, 0.9159075021743774, -0.12908922135829926, 0.21998842060565948, -0.24065393209457397, -0.11322357505559921, 1.9004641771316528, 1.5378412008285522, -0.5631904006004333, 0.04227862507104874, -0.029563268646597862, -0.1671762317419052, -0.07516923546791077, 0.1580316126346588, 1.9578475952148438, 0.5922471880912781, -0.1573566496372223, -1.3883877992630005, -0.4003431797027588, -0.15828000009059906, -0.400410920381546, -0.4252327084541321, 1.1580690145492554, 0.1500837355852127, -0.7583927512168884, -0.2323472946882248, -0.03155096247792244, -1.277799367904663, 0.36547115445137024, 1.1937593221664429, 2.4883475303649902, 0.40588659048080444, 0.5204651355743408, -0.3739115595817566, 0.0072037214413285255, -0.25311508774757385, 1.3893969058990479, -0.5880780220031738, -0.44249534606933594, 0.011666540056467056, -0.0922367051243782, 0.2506047487258911, -0.5811434984207153, 1.915420413017273, 1.3802119493484497, -0.9428728222846985, 0.44467705488204956, 0.20956411957740784, -0.3282218277454376, 1.331222414970398, -2.8830411434173584, 1.1890108585357666, -0.11160908639431, -0.5796886086463928, 0.16727373003959656, -0.5731884837150574, 1.0105211734771729, 0.15981394052505493, -1.1104090213775635, -0.033883530646562576, -0.6494989395141602, 0.22356677055358887, -0.2108006328344345, 0.010340878739953041, 0.34455302357673645, 0.21737810969352722, 1.1186374425888062, -1.4943015575408936, -1.0448359251022339, 0.9282116889953613, 0.0006501668831333518, 0.20399896800518036, 0.6381292343139648, -0.11485899239778519, 0.06392938643693924, 1.4475690126419067, -0.8081983923912048, 0.9116093516349792, 0.07961849868297577, 0.49949297308921814, 2.316359043121338, -2.24896502494812, 0.22923614084720612, -1.628381609916687, 0.6621878147125244, -0.26850736141204834, -0.3078700006008148, -0.33401185274124146, 0.19748598337173462, 1.8458209037780762, 0.0997568890452385, -0.17148911952972412, 0.9529344439506531, -2.181621551513672, 0.7179952263832092, 0.9140253663063049, -0.061585891991853714, -1.5712013244628906, 0.42710062861442566, -1.300183653831482, -0.559147834777832, 0.14693786203861237, 0.06360608339309692, -1.3036099672317505, -1.1140739917755127, -0.24931396543979645, -0.8382164239883423, 2.0169808864593506, -2.204275369644165, 1.9060624837875366, -0.46477147936820984, -0.8879606127738953, 1.96250319480896, -0.6879220008850098, -2.2425220012664795, 0.35121914744377136, 3.627737522125244, -0.8696690797805786, -0.5717325210571289, -1.378007411956787, -1.4498438835144043, 1.0749458074569702, 0.6447619199752808, 1.0889627933502197, 1.1603130102157593, 0.1840367317199707, 1.4979249238967896, 0.6838192343711853, 1.5573724508285522, 2.2470290660858154, -1.1405141353607178, 0.19455543160438538, 1.8210091590881348, -0.7241873741149902, -0.04073546826839447, 1.2413265705108643, 0.17816759645938873, -0.3943158984184265, 0.7296911478042603, -0.12013297528028488, -0.30021440982818604, 2.187368392944336, -0.48408252000808716, -1.2271019220352173, 0.9225566387176514, 0.9312795996665955, -0.523921012878418, -0.7259451150894165, -2.140226125717163, -1.0408738851547241, 0.20866145193576813, 0.461703896522522, 0.3186367452144623, 0.5076470971107483, -0.31801509857177734, -0.2710128128528595, -0.7579447627067566, -1.0782681703567505, -0.03299920633435249, 0.751589834690094, 0.4095655381679535, 0.08102723956108093, -0.3023606538772583, 1.8837146759033203, 2.2616665363311768, 2.931140661239624, -0.2452077865600586, -0.7313075065612793, 0.59479820728302, -1.3935421705245972, -0.5168461799621582, -1.329643964767456, 0.5579270124435425, 0.2384743094444275, 0.7304551005363464, -1.5973185300827026, 0.9061810374259949, 1.0519434213638306, -0.4817047417163849, 0.7583901882171631, -1.4108482599258423, -0.02903810143470764, 0.5084954500198364, -1.7146358489990234, 0.3517049252986908, -1.5013147592544556, 0.6597713828086853, -0.48907220363616943, 0.2805868983268738, 1.5159937143325806, 0.6411525011062622, 0.44341644644737244, 0.7844433188438416, -0.09296903014183044, 0.7689425349235535, -1.368511438369751, -0.789231538772583, 0.35286733508110046, 1.2072861194610596, -0.30604633688926697, 0.5545979142189026, 0.2139596939086914, 0.4870000183582306, -0.9290000200271606, -0.5084819197654724, 1.0639488697052002, -1.6408371925354004, -0.35560372471809387, -0.9168481826782227, -1.6527734994888306, -0.02685185894370079, 1.0439867973327637, 0.8620189428329468, 0.045600347220897675, 1.7174218893051147, 0.45008882880210876, 1.0272170305252075, -2.0386292934417725, -1.6673130989074707, -0.29833170771598816, -1.1876721382141113, -0.4725767970085144, 0.7690556049346924, 0.15867336094379425, -0.7490260601043701, -0.5409997701644897, 0.33987003564834595, 0.9307732582092285, 0.550728976726532, -1.151737093925476, -1.5656408071517944, -0.21941669285297394, 0.4731220304965973, -0.533237099647522, 0.3415082097053528, 0.19287294149398804, 0.7007794380187988, -0.3949173390865326, -0.5142050981521606, -1.5837088823318481, -0.5732007622718811, 0.026388058438897133, -0.637157142162323, -0.6148571372032166, 0.16498900949954987, 0.05415107682347298, 1.7028652429580688, -2.183948516845703, -0.15139085054397583, -0.8551076054573059, -0.9625539779663086, -0.16918671131134033, -0.8663279414176941, -0.33600813150405884, 0.6786494255065918, 0.23741760849952698, 0.6133517026901245, 0.30598682165145874, 1.1888302564620972, 1.665144920349121, -0.03210890293121338, 0.021685220301151276, 0.8034784197807312, 0.36027973890304565, -0.8435544967651367, -1.1499276161193848, -2.1259047985076904, -0.808702290058136, -0.5231635570526123, 0.9435296058654785, -0.9672547578811646, 1.662487268447876, 0.041488803923130035, -0.12303246557712555, 1.4712815284729004, -0.40078040957450867, -0.11000479012727737, -1.2638404369354248, -0.9047114849090576, -0.8055544495582581, -1.1416592597961426, 1.8629449605941772, 1.511273741722107, 1.7017695903778076, -0.8905463814735413, 0.8700070381164551, 0.32065626978874207, 2.7985873222351074, 0.1459076851606369, 0.1917068511247635, -0.7522904872894287, -0.6533335447311401, 0.16440242528915405, 0.3633659780025482, -0.4609757661819458, -1.400504469871521, 0.35712119936943054, -0.25940418243408203, 0.4737145006656647, 0.9338833093643188, 0.7724318504333496, -0.06708225607872009, 0.4952459931373596, -0.28617429733276367, 0.7082561254501343, 0.07877453416585922, -0.31095701456069946, -0.5705711245536804, 0.30292171239852905, -1.8687349557876587, -0.19808566570281982, -1.0769436359405518, -0.1455991417169571, -1.381029486656189, -0.9684381484985352, 1.504325270652771, 0.395975261926651, -0.41058680415153503, -1.1205580234527588, 0.8615391254425049, 0.525346040725708, 1.9254523515701294, 0.2509791851043701, 2.6271049976348877, 0.7926610112190247, -0.9628801941871643, -1.9169631004333496, 0.1610996276140213, 0.3005457818508148, 1.2589704990386963, 0.08583669364452362, -0.2887416183948517, -2.012270450592041, -0.34811675548553467, 1.1365559101104736, -0.7742723822593689, 0.9115824103355408, -0.5087295174598694, 0.809093713760376, -1.6529943943023682, 0.7184997797012329, -0.8412322998046875, 0.7258134484291077, -1.1027584075927734, 0.2164434939622879, -0.13531750440597534, 0.34706392884254456, 1.7697675228118896, 0.3835670053958893, -1.6253187656402588, -1.8860615491867065, -1.0888196229934692, 0.2903425097465515, 0.42336878180503845, -0.2135288417339325, -1.2873852252960205, 1.7795428037643433, -1.7308789491653442, 0.6330610513687134, 0.5181530714035034, 0.217874214053154, -0.2981520891189575, 1.1330422163009644, 0.7370728254318237, 0.16203391551971436, 0.48833975195884705, -0.49689024686813354, 0.11085750162601471, 0.29571333527565, 1.8100860118865967, 2.028785228729248, 0.17019908130168915, -0.45760849118232727, -1.3781872987747192, -0.39272430539131165, -0.6874638199806213, -0.06665860116481781, 0.8735645413398743, -0.7846486568450928, -0.4334392249584198, 0.025639766827225685, 1.3925987482070923, 1.6558551788330078, -0.357216477394104, -0.007579767610877752, 1.7995028495788574, 0.32760077714920044, 0.8317732810974121, 1.2050477266311646, -1.3310834169387817, 0.22753562033176422, 0.5352389812469482, -0.7968279123306274, 0.13915608823299408, 0.9588450193405151, 0.5244337916374207, -1.4437150955200195, 0.20182697474956512, -0.5956143140792847, 2.489964723587036, 0.18503393232822418, -0.13308502733707428, -2.1774163246154785, 1.2657626867294312, 0.460316926240921, -1.1828668117523193, -1.3416757583618164, 0.556540846824646, 0.3262332081794739, -1.2410422563552856, -0.5558427572250366, -1.8994946479797363, 2.4106500148773193, -0.8211919069290161, -1.0774263143539429, 0.5194622874259949, -1.3596134185791016, 0.8428462147712708, 1.0255401134490967, -0.199771910905838, 1.419551968574524, 1.108720302581787, 0.27860718965530396, -1.0028637647628784, 0.8086560964584351, 1.4053651094436646, 2.11641526222229, 0.2437870353460312, 0.5005917549133301, -0.35609543323516846, -0.9374675154685974, -0.9752206802368164, -1.969866156578064, -0.1368388533592224, -0.562176525592804, 0.2542543113231659, -0.5031859874725342, -1.4638556241989136, 0.6365604400634766, 0.3207970857620239, -0.5558109283447266, -1.002087950706482, -1.6820096969604492, -0.5260744690895081, 0.6352343559265137, 0.1402612030506134, -1.769300937652588, 0.6857671141624451, -1.0664639472961426, -0.4717983603477478, -0.3841804563999176, 0.797197163105011, 0.7822736501693726, 0.2572118639945984, 0.11087687313556671, 0.6175982356071472, 0.46177995204925537, 1.187298059463501, 0.08659816533327103, -1.1729823350906372, -2.0901405811309814, -0.3795221745967865, 1.7412114143371582, 0.26461246609687805, 0.4652201533317566, -1.3121353387832642, 1.0380254983901978, 2.597543954849243, -0.7445646524429321, 1.399261236190796, 1.3900002241134644, 0.1431349217891693, 0.6797052621841431, 1.1876776218414307, -2.0603959560394287, -0.14637456834316254, -0.08308953791856766, 0.9130294919013977, -1.47456955909729, -0.6691784858703613, 0.17480075359344482, 0.4542642831802368, 0.6054624915122986, -0.11034996807575226, -0.6984232068061829, 0.5094543695449829, 0.46300238370895386, 0.345020055770874, -0.0856102705001831, -0.007825014181435108, 1.381227731704712, 0.30675604939460754, 0.020143771544098854, -0.9527438879013062, 0.15501698851585388, -0.5998312830924988, -0.7941089868545532, -0.7041411399841309, 1.6951323747634888, 1.9429826736450195, 1.2455270290374756, -0.040829382836818695, -1.5315794944763184, -0.7916182279586792, -0.47783342003822327, -0.6620965003967285, -0.6338781118392944, -1.6855324506759644, 0.3398136794567108, -0.12659363448619843, -0.4758748412132263, 0.8698447346687317, -1.1194566488265991, 0.9325888752937317, -1.1462650299072266, 0.6219377517700195, 1.5728458166122437, -0.16284030675888062, 0.1627146452665329, -0.7534947395324707, -0.7012467384338379, -0.21317125856876373, -0.7301437854766846, -0.5057248473167419, 0.5389395952224731, 1.5918307304382324, -0.38306087255477905, 0.9528287053108215, -2.315857172012329, -0.15089461207389832, -0.6555688977241516, -0.26958104968070984, -0.9246644377708435, 0.8960596919059753, 0.0805899053812027, -0.7932085394859314, 0.6190568208694458, -1.4174299240112305, -0.39215776324272156, 0.24435968697071075, -0.664158046245575, -1.5353459119796753, -0.9957725405693054, 0.300642728805542, 0.7113597989082336, -1.6034762859344482, -0.06405618041753769, -0.15055543184280396, 0.6344390511512756, 0.4721619784832001, 0.018798086792230606, -0.2762162983417511, -0.640083372592926, -0.8898214101791382, 0.36822783946990967, 0.4791692793369293, 0.11018558591604233, -1.2888950109481812, 0.5045921802520752, -1.1768115758895874, -0.06151817739009857, 0.4015088975429535, -0.6258651614189148, -1.9532768726348877, 0.4569986164569855, 0.642974853515625, -1.747747778892517, -1.5856871604919434, -1.1780524253845215, 1.7988978624343872, -1.7252854108810425, 0.8939113616943359, -1.473726511001587, -0.2922601103782654, -0.1454886794090271, -0.1338992863893509, 0.20970511436462402, -0.44796594977378845, -0.14501328766345978, -1.2722208499908447, 0.6755779981613159, -0.71805340051651, 0.5909969806671143, -0.6955104470252991, -0.09683723002672195, 0.06568198651075363, -0.20368905365467072, 1.960911512374878, 0.04391491785645485, 1.416911244392395, -0.4687073826789856, -0.8462892770767212, 0.10134954750537872, 1.5868104696273804, -0.4773404896259308, -2.38598895072937, 0.43666887283325195, -0.9966485500335693, -0.6380123496055603, 0.005154345650225878, -0.5403735041618347, -0.6605890989303589, -0.2650470435619354, 1.0652350187301636, 0.24963690340518951, 2.2020833492279053, -0.45549526810646057, 1.00849187374115, -0.10108744353055954, 0.05193477123975754, 0.28091365098953247, 1.2111626863479614, -1.296992301940918, 0.342793732881546, -0.06360586732625961, 0.07417160272598267, 0.2753101885318756, 0.12387563288211823, -1.0047013759613037, -0.40839213132858276, 0.5137407779693604, -1.3198095560073853, 1.5762196779251099, -1.317099928855896, 0.14969338476657867, -1.4566588401794434, -0.4445415735244751, -0.6181067824363708, 0.6130868792533875, 0.08033449947834015, 1.6930214166641235, -1.124083161354065, -1.3841301202774048, 0.6763380169868469, -1.1473863124847412, 0.32219740748405457, -0.6559640169143677, 0.4390438199043274, 0.5306611657142639, 0.26290732622146606, -0.8971495628356934, -0.09311731904745102, -0.5306078195571899, -0.9088388681411743, -0.38245120644569397, -1.1059643030166626, 0.24725116789340973, 0.21906308829784393, 0.4713574945926666, -0.726536750793457, -0.37035778164863586, 0.582802951335907, 0.762848436832428, -0.8689588904380798, 2.061244010925293, -0.03767344355583191, -0.03474345803260803, 0.7964447736740112, -1.2207109928131104, -1.124069094657898, -0.31342318654060364, 0.3486609160900116, 1.7360968589782715, 0.8793200850486755, 1.0551658868789673, 0.4837478995323181, -1.3806393146514893, -0.2845211327075958, -0.06701777130365372, -0.1911858767271042, 0.09859068691730499, 0.05078396946191788, -0.8296890258789062, -0.28436800837516785, 1.4472389221191406, -1.1264843940734863, -0.2934589684009552, 1.0901564359664917, -1.2874869108200073, 0.16065756976604462, 1.8171275854110718, -0.9141634702682495, 1.7754836082458496, 2.4843947887420654, -2.0629401206970215, -0.05886680632829666, -0.3867360055446625, 0.14269079267978668, -1.4502441883087158, 0.20303218066692352, -1.2901396751403809, -0.5747758746147156, -1.025612473487854, -1.881793737411499, -0.19382387399673462, 0.45572343468666077, 1.4201463460922241, 0.6329796314239502, -1.485129475593567, 1.1783232688903809, 0.6746526956558228, 0.38497301936149597, -0.7380079030990601, 1.00190269947052, -0.7401423454284668, -1.079488754272461, 0.7170252799987793, -0.07857859879732132, -0.9389567375183105, -0.5195664763450623, 0.701495885848999, -0.5337162017822266, 0.1298314929008484, 0.4287329912185669, -1.8545477390289307, 0.7055374979972839, 0.10523569583892822, -0.3853829503059387, -0.23247209191322327, 0.13071583211421967, -1.3785321712493896, 0.33667340874671936, -0.13060417771339417, 0.5104408860206604, -0.7072336077690125, -0.938389778137207, -0.713965117931366, 0.38321635127067566, 0.6149458289146423, 0.11427396535873413, -0.3054761588573456, -0.34605807065963745, 0.32644742727279663, 1.0781651735305786, -0.3157249689102173, -0.6875144839286804, -1.2959829568862915, -0.10944479703903198, 0.401013046503067, -0.4059409499168396, 0.07592017203569412, 0.43018919229507446, -0.5176779627799988, -0.9039760231971741, 0.17540620267391205, -1.5306638479232788, 0.14481788873672485, 0.22515934705734253, -1.1156857013702393, 2.633371114730835, -1.0207449197769165, -0.1874372512102127, 1.2267355918884277, -1.0300627946853638, -0.2340661585330963, -0.9971511960029602, -0.2131732851266861, 1.8810703754425049, 0.7388444542884827, 0.994635820388794, 0.11865450441837311, 0.6382951736450195, 0.2856702208518982, -0.004037931095808744, -1.0665555000305176, 0.7812321186065674, 0.22695551812648773, 1.8933217525482178, 0.5707364082336426, 0.1017032265663147, -0.611582338809967, 0.13122797012329102], [1.1902235746383667, 0.5747944712638855, -0.24360033869743347, -0.9587255120277405, -0.5684343576431274, 0.2464764267206192, -0.6900418996810913, 0.16579346358776093, 0.6611282229423523, 0.7946398258209229, 1.4756985902786255, 0.3943776488304138, -1.208733081817627, -0.33147212862968445, 0.629144012928009, -0.5769867897033691, 0.29161545634269714, -0.35408538579940796, -1.2721349000930786, -0.5526251792907715, 0.7420136332511902, 1.3459700345993042, 0.23209095001220703, -1.3110826015472412, 0.9771586656570435, 1.369421362876892, -0.4615723788738251, 0.2398330569267273, 0.5396226644515991, -0.7762488126754761, 0.8807230591773987, 1.658313512802124, -0.9148541688919067, 1.102689504623413, -1.1311676502227783, 1.1264134645462036, -1.6398892402648926, -0.7785601019859314, -1.3031781911849976, 0.1681383103132248, -0.36505499482154846, -0.44386255741119385, -1.8450571298599243, -1.0241950750350952, 0.7571508288383484, -0.21215584874153137, -1.368341326713562, -0.46192577481269836, -1.0539100170135498, -0.40416887402534485, 0.9054533243179321, -0.40324026346206665, 0.818225622177124, -0.4581132233142853, -0.02785707451403141, 0.2748202681541443, -0.697113573551178, -0.855266273021698, -0.8759143352508545, 1.1126060485839844, 1.3616029024124146, -0.7068402767181396, 1.0054869651794434, 0.8331509828567505, -0.26701614260673523, 0.569574236869812, 0.6686837673187256, -0.46909332275390625, -0.3903917372226715, -0.8146661520004272, 1.7087174654006958, -0.6914671659469604, -0.17155498266220093, 1.820685625076294, -1.3012335300445557, 0.24206581711769104, -0.8660659790039062, -0.4425639510154724, 0.43535929918289185, 0.5804360508918762, 0.34682461619377136, 0.6963949203491211, -1.015416145324707, 1.7578916549682617, -0.5226231813430786, -1.1038165092468262, -1.1295218467712402, -0.6498836874961853, -1.4602653980255127, 1.61117684841156, 1.9304431676864624, -0.6054866909980774, 0.9662561416625977, -2.1305935382843018, -0.16268691420555115, 2.085862636566162, 1.234375238418579, 1.1830527782440186, 0.5900031328201294, 0.01995493471622467, 0.8079259991645813, -0.3981880247592926, -0.5682088732719421, -1.8129546642303467, 0.08807410299777985, 0.6794403195381165, -0.21514460444450378, 0.1317787766456604, 2.0498595237731934, 2.776712656021118, -0.3436267375946045, -0.5214059352874756, 0.05838451534509659, -1.1842297315597534, 1.4089902639389038, -0.34696316719055176, 0.12887123227119446, -1.2678879499435425, 0.8659253716468811, 0.06039771810173988, -0.4502485394477844, -1.1169146299362183, 0.7356662154197693, -0.8981021046638489, 0.747097909450531, -0.36554595828056335, -1.6892757415771484, 0.476809561252594, -0.29652875661849976, 0.19203954935073853, 0.25889816880226135, 0.8474143743515015, 0.11349344998598099, 0.6649444699287415, -0.22950464487075806, -0.35151299834251404, -0.12335648387670517, 0.5576815605163574, -0.5248135924339294, 0.19206340610980988, -0.011626075021922588, 0.3553416132926941, -0.5740655660629272, -0.30867862701416016, -2.2486772537231445, 0.2073974460363388, -1.7828317880630493, 1.5677337646484375, -0.9886025190353394, -0.8960124254226685, 0.6680005788803101, 0.2461382895708084, -0.7445763945579529, 0.5822617411613464, -0.5056825876235962, 0.6722239255905151, -0.4131165146827698, -0.6084996461868286, -0.27689793705940247, -0.0044466471299529076, -2.3919119834899902, -0.21120551228523254, 1.025524616241455, -1.0069522857666016, -0.513047993183136, -0.15612103044986725, 0.19119282066822052, -0.6797890663146973, 0.02390330098569393, 0.9558054208755493, 0.6937597393989563, -0.786353588104248, 1.4451814889907837, 1.078763484954834, 1.412026047706604, 0.1345333755016327, -1.8011701107025146, -0.6759440898895264, -0.8729252815246582, 0.2046366184949875, 0.037658460438251495, 1.6592636108398438, -2.966524124145508, -2.108121871948242, 0.1710817664861679, -0.37975823879241943, -0.18909917771816254, -0.4262531101703644, -1.446171760559082, -1.6661607027053833, -2.156379222869873, 0.4721156060695648, 1.1494182348251343, -0.161092147231102, 0.4010829031467438, -1.1543755531311035, -1.535483717918396, -0.40977662801742554, 0.2456977367401123, 0.7981422543525696, -0.6526065468788147, 0.10519584268331528, -0.3233339190483093, -0.11346423625946045, 0.3722156286239624, -0.8759821653366089, 0.2399914264678955, -0.6236361265182495, 0.15824919939041138, 0.2596992552280426, 0.9950228929519653, -1.4166138172149658, 0.8359430432319641, -1.0341227054595947, 0.10291711986064911, 0.4269760847091675, -0.3402780294418335, 0.6071783900260925, -1.5294642448425293, 1.1732566356658936, -2.241105794906616, 0.47302642464637756, 0.6930933594703674, -1.305732011795044, -1.9947330951690674, -1.0455491542816162, -0.6654868721961975, 0.8823822736740112, 0.6022775769233704, -1.0904160737991333, 1.2829086780548096, -0.3340533971786499, -0.047932107001543045, -0.46827271580696106, 1.5082571506500244, 0.6244199872016907, 0.324687659740448, 1.6060768365859985, -1.1223865747451782, 0.2756575047969818, 0.46039271354675293, -0.86381596326828, 0.8020018339157104, -0.56241774559021, 1.3326884508132935, -0.41706976294517517, 0.7350170612335205, 0.6273891925811768, 1.3536832332611084, 1.5203399658203125, -0.18646861612796783, -0.9967857003211975, -0.017370810732245445, -1.6110765933990479, 1.1897729635238647, -1.6055773496627808, -0.02809281088411808, -1.058816909790039, -2.022080421447754, -0.4470212459564209, -0.14048738777637482, -0.6391631960868835, 2.0967507362365723, 1.5568443536758423, 1.2337253093719482, -1.6132807731628418, -1.609579086303711, -0.7776489853858948, 0.6091985702514648, -0.26382115483283997, -0.19904825091362, 0.7012497186660767, 0.1743498593568802, -0.37216293811798096, 0.9151118397712708, 0.49505671858787537, -0.3532380759716034, -2.2457282543182373, -0.8217307329177856, -0.28115397691726685, -0.72893887758255, 0.01130871381610632, 0.18462026119232178, -0.9338833689689636, -0.8876618146896362, 1.2265735864639282, -1.2440227270126343, -1.1043922901153564, -1.0024065971374512, -0.04500254616141319, 0.10615359991788864, -1.121300220489502, -0.5557133555412292, -0.4100337326526642, -0.5870169401168823, -0.2788204252719879, -0.48343425989151, -0.9777818322181702, 1.1554564237594604, -0.7751314043998718, -1.6747844219207764, 0.3455013334751129, 0.7967052459716797, 1.2697936296463013, -1.3234148025512695, 1.065812587738037, -1.226073145866394, 0.15086649358272552, 0.37767845392227173, -0.7152026295661926, 0.9971361756324768, 0.7001572251319885, 0.04325088486075401, -3.121363639831543, 0.3412083387374878, 0.6875025629997253, 0.8309241533279419, 0.9223670363426208, 0.907683789730072, -0.3332395553588867, 0.9918571710586548, -0.4824689030647278, 0.8368767499923706, -0.39627328515052795, 0.8098159432411194, 1.4291843175888062, 0.569908618927002, -0.41649383306503296, 0.9459810853004456, -0.8761036396026611, -1.0799646377563477, 1.1251685619354248, 1.3390377759933472, 0.6400336027145386, -2.124922752380371, -0.42666327953338623, 1.5705163478851318, 0.5600337982177734, -0.03674599900841713, -1.4007632732391357, -0.16376876831054688, -1.5242416858673096, 0.8498960733413696, 0.7330210208892822, 1.132137417793274, -1.1285829544067383, -0.3270266354084015, -0.40321364998817444, 1.4411314725875854, 0.7210195660591125, 0.6020801663398743, -1.8292266130447388, 1.710839867591858, -1.7651731967926025, -0.627577006816864, 1.5804588794708252, 0.1825975775718689, -1.3818507194519043, 0.7421836853027344, 0.5544222593307495, -0.022233422845602036, 0.651516318321228, 1.2364017963409424, 0.6176394820213318, -0.5424120426177979, -0.22355812788009644, -0.4978514015674591, 2.116084575653076, 2.103398323059082, 0.07683195918798447, -1.3495818376541138, -0.45188790559768677, 1.436597466468811, -0.2523880898952484, -1.1677353382110596, -0.19747184216976166, -0.3807573616504669, -0.2911970615386963, 0.9836475253105164, 0.6431991457939148, 0.6208361983299255, 0.2796463668346405, -0.19815589487552643, -0.48030272126197815, -0.4207000434398651, -0.05441911891102791, 1.1889925003051758, -0.7041152119636536, -0.5487968325614929, -0.6976877450942993, -0.7936599850654602, -1.1378237009048462, -1.0338612794876099, -0.8038533926010132, 0.4760318696498871, 0.918610692024231, -0.4653061330318451, -1.0921626091003418, -0.3552524745464325, 0.18988703191280365, -0.4283785820007324, 0.31560713052749634, -1.0835998058319092, 0.6777295470237732, -0.02678261697292328, 0.253875195980072, -0.5679185390472412, -0.0007752582314424217, 1.9413316249847412, -0.09677498042583466, 1.1204288005828857, -0.6857331991195679, -0.6964268088340759, 0.5297709703445435, -0.38247808814048767, -1.5538403987884521, -1.0743643045425415, -0.024336790665984154, -1.859121322631836, -0.2707379460334778, 0.8266912698745728, -1.217356562614441, 0.656599760055542, -0.9824520945549011, -0.8702995777130127, 0.46519801020622253, 0.00961392279714346, -0.6709080338478088, 1.4414832592010498, 0.8024030327796936, -0.19485855102539062, 1.5563594102859497, -0.33780738711357117, 0.47770553827285767, -1.8437012434005737, -0.7874869108200073, -2.3640072345733643, 0.19267141819000244, -0.40735796093940735, -1.4791572093963623, 0.9286791682243347, 0.18566223978996277, -0.4983929991722107, -0.4334769546985626, -0.6984464526176453, 0.7614215016365051, 0.7689164876937866, 0.18741151690483093, -1.114302158355713, -0.8941581845283508, 2.0071253776550293, -1.228147268295288, -0.17890849709510803, 0.19879557192325592, 0.524647057056427, -0.48201802372932434, 1.518117070198059, -0.7242276668548584, 1.0429046154022217, -0.0814141035079956, -0.3991059958934784, 1.6151586771011353, 0.31340789794921875, 1.994323968887329, 0.2997724413871765, -2.4851198196411133, -1.3218269348144531, 0.7374837398529053, -1.4387563467025757, 0.05096020922064781, 2.2037253379821777, -0.13092246651649475, 0.4072614312171936, -1.1102409362792969, 0.09332935512065887, 0.3819585144519806, -0.2089867889881134, -0.12279108911752701, 0.5936552286148071, 1.6152251958847046, 0.5231591463088989, -0.37909984588623047, 0.03987288475036621, -0.1582973748445511, 0.40414756536483765, 1.8246129751205444, -0.39093658328056335, 0.7031048536300659, 2.1460258960723877, -0.38192394375801086, 0.4412463903427124, 0.6647631525993347, 0.16262155771255493, 1.2008752822875977, -1.426716685295105, -0.4395281672477722, 2.2651913166046143, -0.5053271055221558, -0.13558073341846466, 0.9847449064254761, 0.2600450813770294, 1.7194501161575317, -0.2943708002567291, -0.6097167730331421, -0.19865967333316803, 1.2427548170089722, 1.2056198120117188, -1.0001161098480225, 1.2369180917739868, 0.14190158247947693, 1.0393610000610352, -1.542055606842041, 0.8161142468452454, -2.0174577236175537, -0.323553204536438, 2.3265554904937744, -0.4732672870159149, 2.7518625259399414, 1.1103907823562622, -0.6050951480865479, -0.9116552472114563, -0.33009403944015503, -0.9246900677680969, -0.10803785175085068, -1.022653341293335, -1.0401605367660522, 0.26924628019332886, -0.16735069453716278, 1.0012753009796143, -1.3699806928634644, 0.3510757088661194, -0.4050557613372803, 0.9632086753845215, 1.5718709230422974, -1.0420466661453247, -0.7851600050926208, -1.5483570098876953, 0.1637636125087738, -0.7656459212303162, -0.9288135766983032, 1.2875969409942627, -1.097743272781372, 1.7143570184707642, 1.602413535118103, 0.16775880753993988, 1.5416061878204346, -2.085564136505127, 1.4181214570999146, 0.9553294777870178, -0.8936880826950073, -0.30810779333114624, 1.495080828666687, 0.354472279548645, -1.166285514831543, 0.4984378218650818, 1.1358104944229126, -1.7182905673980713, -0.004882684908807278, -0.5446481108665466, 0.10905346274375916, 0.15297964215278625, 0.7615521550178528, 0.06878502666950226, 0.1670197695493698, -0.4602719247341156, 1.084848165512085, -0.17812342941761017, 0.9365497827529907, 0.00944148562848568, -0.061044126749038696, -1.5739856958389282, -1.8459476232528687, -1.5263928174972534, -1.4866573810577393, 0.5574802160263062, -0.9040443301200867, -0.7720789313316345, -2.1734442710876465, -0.029035232961177826, -1.3227601051330566, -0.4079190790653229, 0.3277168273925781, 0.7936312556266785, -0.024827782064676285, -0.20746776461601257, -0.4174661636352539, -1.3681421279907227, -1.4917932748794556, -2.343975067138672, -0.822160542011261, 0.1888851672410965, -0.8451824188232422, 1.1273980140686035, -0.18900741636753082, 0.9548231959342957, -0.4436682164669037, 0.6621017456054688, 0.8206714391708374, 0.7312377691268921, 0.5879644155502319, -0.4394139051437378, -1.6253206729888916, 1.4866993427276611, -1.060636043548584, -0.5942534804344177, -1.5332069396972656, -1.9394969940185547, 0.004101564176380634, -0.7651523947715759, -0.48831576108932495, -0.3140604496002197, 0.3380797505378723, -0.03390500694513321, -0.5354536771774292, 0.4794374108314514, 0.10224950313568115, -0.6715568900108337, 0.584932267665863, 0.39097148180007935, -0.9585906863212585, 0.028738901019096375, -0.2793874442577362, 0.2530757784843445, 0.8333104848861694, 0.49562790989875793, -0.38189464807510376, 2.2148237228393555, -1.1929435729980469, -1.092942476272583, 0.46510645747184753, 2.0639243125915527, -2.4294815063476562, 0.624596357345581, -0.6643660664558411, 2.3432843685150146, 0.07070950418710709, 0.6166861057281494, -0.3167840540409088, -0.8370764255523682, -0.13685905933380127, -0.41427212953567505, -0.8116198182106018, 0.44897696375846863, -0.5976104736328125, -0.9841558337211609, -0.40922847390174866, 0.6338306665420532, -1.2963101863861084, 0.6719632744789124, 0.4375506043434143, 0.41962316632270813, -1.0677684545516968, -1.4076825380325317, -0.569602906703949, 0.31440672278404236, -0.6770704984664917, -0.6522195935249329, -1.8745261430740356, 1.1149928569793701, 1.5712826251983643, 0.9339702725410461, -0.39231422543525696, -0.887165904045105, -0.04801548272371292, 0.22812502086162567, 1.5865528583526611, -0.6653504371643066, -0.3586321771144867, 0.42973873019218445, 0.3328792154788971, 1.7791048288345337, 0.18532347679138184, -1.233919620513916, -0.9428742527961731, 1.3714625835418701, -1.5219312906265259, 0.1061147153377533, -1.8599789142608643, 0.9809598326683044, -1.3284622430801392, 0.3891923725605011, 0.3536861538887024, 0.15469157695770264, -0.4753989577293396, -0.5090383887290955, -0.39496225118637085, -1.4122811555862427, 0.2270619422197342, 0.7274452447891235, 1.856295108795166, 1.3279527425765991, -0.32278308272361755, 0.5699513554573059, 1.522438645362854, 1.113578200340271, 1.002964735031128, 0.20454010367393494, 0.786055326461792, -0.11851193755865097, -0.13577799499034882, -0.3900814950466156, 0.43087056279182434, 1.2485175132751465, 0.10135918110609055, 0.42897939682006836, 0.6495016813278198, -0.21829377114772797, 1.5046030282974243, -0.4963100850582123, -0.44480207562446594, -2.0528085231781006, -2.8074138164520264, -0.43436238169670105, 1.7996532917022705, -0.32117760181427, -0.15807318687438965, -0.9463273286819458, -0.51230388879776, 1.8361501693725586, 1.3431110382080078, -0.20960716903209686, -0.688351571559906, 1.386757731437683, 0.24164979159832, 1.7311068773269653, -1.6275548934936523, -0.06653560698032379, 0.42724692821502686, 0.9057213068008423, -0.46269896626472473, 0.9443906545639038, 0.5350484251976013, -1.341205358505249, 1.0118956565856934, 0.2508980631828308, 1.2206610441207886, 0.23604898154735565, -1.0458159446716309, 0.7188846468925476, 0.17585672438144684, 1.345104694366455, -0.39766958355903625, 0.21983690559864044, 0.8509592413902283, -1.4825475215911865, 1.2722914218902588, 0.9582825899124146, -0.9270886778831482, -0.48069649934768677, -0.060875747352838516, 1.9933876991271973, 1.8056135177612305, -0.5905956029891968, 1.068753719329834, 0.2502744197845459, -0.030560245737433434, 1.0941742658615112, -0.7670589685440063, -0.301487535238266, -0.7979615926742554, -1.1065983772277832, -0.11698757857084274, -0.2186439335346222, 0.8337017893791199, 1.293083906173706, -1.177272081375122, -0.2083347886800766, 0.7737286686897278, -1.9758553504943848, -0.3274753987789154, -1.1516094207763672, 0.9972200989723206, 1.6655430793762207, -0.5438351631164551, 2.3076529502868652, 0.6482939124107361, -0.7362565398216248, 1.4455986022949219, -0.10102346539497375, 0.11231683194637299, 0.6112411618232727, -0.8403968811035156, 0.0205489881336689, 1.28106689453125, 0.17042171955108643, -0.15901054441928864, 0.20459310710430145, -1.7511392831802368, -0.8073440194129944, -0.9682204127311707, -1.0553326606750488, 0.47923362255096436, -0.10699008405208588, 0.01584308035671711, 0.6046015024185181, -1.00271475315094, -0.9886447787284851, 2.511784791946411, -1.3369700908660889, 2.241115093231201, -0.6382003426551819, -1.874695062637329, 1.0629538297653198, -1.6863069534301758, -0.49990686774253845, 0.9853827357292175, 2.3485829830169678, -1.1532303094863892, -0.885185182094574, 0.6405092477798462, 1.182915210723877, -0.912729024887085, 0.19471313059329987, -0.7706429958343506, -0.32633957266807556, 1.4578115940093994, 1.2131942510604858, 0.24515172839164734, 1.0537606477737427, 0.6726581454277039, -0.0881333127617836, 0.7795836329460144, 0.15391068160533905, -1.0625026226043701, 0.07857698947191238, -0.3920604884624481, 0.8500632643699646, 0.4882483184337616, -1.8093291521072388, 0.8714982867240906, -0.5521239638328552, 1.1077390909194946, 0.5874633193016052, -0.5802490711212158, -0.6966779828071594, 0.17054586112499237, 1.5808929204940796, -0.7514363527297974, 0.3758660554885864, -0.452260285615921, 0.41253000497817993, -1.2470017671585083, 1.6758605241775513, 0.02388407476246357, 0.7087177634239197, 1.486892819404602, -0.5628792643547058, 0.4052092134952545, 0.1180490106344223, 0.12956388294696808, -1.368761420249939, -2.058134078979492, 2.28704571723938, 0.5617527365684509, -0.4354962110519409, -0.3083602488040924, 1.6941763162612915, -0.8888991475105286, 0.6551947593688965, 1.198050856590271, -1.9058337211608887, -1.4143180847167969, 0.7945462465286255, 0.9290289878845215, -1.7724004983901978, -0.8354557752609253, 1.4364738464355469, 0.4685380458831787, 0.3846678137779236, -0.4880133867263794, 1.2634254693984985, -0.7469591498374939, 0.6960402131080627, 0.7599720358848572, 0.8149519562721252, -0.2156280130147934, -0.5028778910636902, 0.09116450697183609, 1.152664303779602, 1.8417879343032837, -0.06930073350667953, 0.38182613253593445, -0.07910538464784622, -1.540632963180542, 0.1755473017692566, 0.44260597229003906, 0.9130545854568481, -0.8152210116386414, 0.10010265558958054, 0.33450230956077576, -1.4882968664169312, -1.0641944408416748, -0.3280330002307892, 0.05328398570418358, 0.05836302414536476, 0.9446073770523071, -0.07143732905387878, -1.4902608394622803, -3.0526394844055176, 0.325875461101532, 1.6963177919387817, 1.9799396991729736, -0.17751747369766235, -0.16671408712863922, 0.6093888282775879, -0.5895797610282898, -1.6834720373153687, 0.5080731511116028, -2.129201650619507, 0.02420641854405403, -1.1884344816207886, -0.39225059747695923, 1.170527458190918, -1.4823707342147827, 0.5043375492095947, 0.9478517770767212, 0.5941406488418579, 1.6515107154846191, 0.466853529214859, -1.7893738746643066, -3.7658512592315674, 1.5673184394836426, -0.42140993475914, 1.373183012008667, 0.7986429929733276, -0.01221463643014431, 0.22082868218421936, -1.563462257385254, -0.4521184265613556, 0.9235726594924927, 1.5567516088485718, -1.5729517936706543, 0.24892477691173553, -1.028070092201233, -0.22705812752246857, -1.4767794609069824, -0.45098254084587097, -1.166527509689331, 0.7076295018196106, -1.5950320959091187, 0.6714293360710144, 0.24867454171180725, 1.3295305967330933, -0.6581189036369324, -0.15774820744991302, 0.6531735062599182, 1.4994287490844727, -0.007617923431098461, 1.2025463581085205, 0.30601537227630615, -0.613445520401001, 1.0293835401535034, 0.6841259598731995, 0.19118477404117584, -0.9775506258010864, -0.9998444318771362, 0.9967020153999329, -2.3788363933563232, -1.631235957145691, 0.5047638416290283, -1.2199079990386963, -0.2584313154220581, -0.9715451598167419, -1.0930044651031494, -0.670783519744873, 1.2662657499313354, 2.576697587966919, 2.7919483184814453, -0.020380493253469467, -0.3685135543346405, -0.03867657482624054, 1.3744369745254517, 2.4687557220458984, 1.1239702701568604, 0.6376426815986633, -0.9177156686782837, 1.3427672386169434, -0.09948261827230453, -0.3434852063655853, -0.8724279403686523, -1.5426362752914429, 0.24298523366451263, 0.5518903732299805, 0.12467776983976364, -0.5733843445777893, 0.2302187830209732, 0.15426217019557953, 0.4275002181529999, 0.0266913753002882, -0.5210059881210327, 0.4126231372356415, -0.6409311890602112, 0.8290807604789734, -0.6439689993858337, -1.2232998609542847, 0.8336390256881714, -0.8742449879646301, -0.6392521858215332, 2.0954856872558594, 1.3543801307678223], [1.4397354125976562, -0.8706349730491638, -0.9737745523452759, -1.812453031539917, 0.4194270670413971, 0.19621604681015015, 1.3244876861572266, 1.1085238456726074, 0.08407490700483322, 0.5146877765655518, -1.1462606191635132, 0.8760949969291687, 0.05674251541495323, 0.3489018380641937, 1.6163537502288818, 2.112840414047241, -0.21773286163806915, -0.42796072363853455, 0.19710198044776917, 1.3697335720062256, 1.1114556789398193, 0.15338867902755737, -1.9308704137802124, -0.8472393751144409, 0.5704968571662903, -0.9601795673370361, -0.7234160304069519, -0.6728836297988892, 0.42019590735435486, 0.8301819562911987, 0.06410478800535202, -0.3217143714427948, -1.0133556127548218, 1.7564295530319214, -1.2768338918685913, -0.185637965798378, 0.015917116776108742, -0.1835862547159195, -0.6799030303955078, -0.4617069661617279, 1.335548996925354, 1.043562650680542, -3.2843546867370605, -1.079561710357666, 1.4104962348937988, -0.6814801692962646, 0.883777916431427, 0.7566642165184021, -0.12829826772212982, -1.665806770324707, 1.4922610521316528, -0.19385254383087158, -1.9849867820739746, 0.7045502066612244, 1.720441460609436, 1.243420124053955, 0.7914566993713379, -0.6196860671043396, 0.25776511430740356, -1.9355180263519287, -0.2196059674024582, -0.22872485220432281, -0.8780545592308044, -0.7006826996803284, -2.113126277923584, -0.22231313586235046, 0.9357349872589111, 0.4741362929344177, -1.0210872888565063, -1.098815679550171, -1.0560753345489502, -0.25756123661994934, 0.9287359714508057, -0.2731616199016571, -0.9192843437194824, -0.15409359335899353, 0.012818911112844944, -0.7344950437545776, 0.09223917871713638, 1.1507230997085571, 0.7071561217308044, 0.5744553208351135, 0.5343706011772156, -0.09838826954364777, 0.101339191198349, 0.470185250043869, -1.9435038566589355, 0.49572062492370605, -1.5491129159927368, -0.6931424140930176, -1.0522842407226562, 1.3431943655014038, -0.2190137803554535, 0.05675671994686127, 0.06255507469177246, 0.7359651327133179, -1.6707199811935425, -0.38051891326904297, -0.618172287940979, 0.5575100183486938, 1.1364349126815796, -0.8716016411781311, -2.318296194076538, 0.701002836227417, 0.8983188271522522, -1.9067515134811401, 0.2823009192943573, -0.9521823525428772, -1.7317304611206055, -0.4050501585006714, -1.80078125, -0.02555478736758232, 0.8789214491844177, 2.198382616043091, -0.16923733055591583, -1.3083873987197876, 1.0311267375946045, -0.18829017877578735, -0.6558609008789062, 0.7591730952262878, 1.7243751287460327, -0.8131071329116821, 0.05768442898988724, 0.07209953665733337, -1.0529201030731201, -0.23405171930789948, -0.7749965786933899, 1.143842101097107, -0.25880154967308044, 0.3187861740589142, 1.7028712034225464, -0.3556702435016632, 0.929863691329956, -0.18992021679878235, -0.5229595303535461, -0.38188260793685913, -0.4068056643009186, -1.251605749130249, 1.503912091255188, 0.03410235419869423, -0.9149637818336487, -1.146633267402649, -0.5244709849357605, -0.8465142250061035, -0.2770571708679199, 1.0821373462677002, -0.8124006986618042, -0.12619511783123016, -0.4395911693572998, 2.211320638656616, -2.07657790184021, -0.10284964740276337, 1.281982183456421, -0.4975084364414215, -0.9243910312652588, -0.5156654715538025, -0.9719838500022888, -1.466091275215149, 1.2959823608398438, 1.1634336709976196, 0.10902789980173111, -0.18917134404182434, 0.4869137108325958, 0.04978800565004349, 1.0161871910095215, 0.12348908931016922, 1.54014253616333, -0.770555317401886, 1.2678135633468628, 1.0185458660125732, 2.3104569911956787, -0.23022682964801788, -1.3725764751434326, -0.2545004189014435, -0.16883717477321625, -1.03728187084198, -0.3173620104789734, -0.6857969164848328, -0.4921824634075165, -1.1284576654434204, -0.8793126344680786, -1.038657546043396, -0.5084008574485779, 1.8380149602890015, -0.6007341742515564, 1.422700047492981, -0.05007980391383171, -0.3196339011192322, -0.6445457339286804, 0.02572776935994625, 1.068105697631836, -2.2717509269714355, 0.06312049180269241, 0.3699412941932678, -0.5898491144180298, -0.26461702585220337, -0.4595358669757843, 0.3532011806964874, -0.8310573101043701, 1.9038547277450562, 1.1149308681488037, 0.6391743421554565, 0.4446495473384857, -0.170400008559227, -1.9502960443496704, -0.8622933030128479, 0.48301032185554504, 1.459468960762024, -0.6806657910346985, 1.616370677947998, -0.09146463125944138, -0.4832438826560974, -1.2795029878616333, 0.4228643476963043, -0.7529640197753906, -1.5049686431884766, 0.662612795829773, -0.2957421541213989, 1.336228370666504, -1.043574571609497, -0.701206386089325, 0.058504290878772736, 0.16482453048229218, -0.10165740549564362, -0.82904052734375, 0.683095395565033, -0.7296502590179443, -0.8040647506713867, -1.4153919219970703, 2.292126178741455, 0.5098273158073425, 0.9226753115653992, -0.09143473207950592, -0.36929917335510254, -0.13357681035995483, 1.4424834251403809, 0.2694839537143707, 0.47837692499160767, -0.9484359622001648, 0.9779872894287109, 1.1457903385162354, 0.23691712319850922, -0.629906415939331, -0.013514265418052673, 0.12155941873788834, -0.8903257846832275, 1.7577933073043823, -1.7453080415725708, -0.9750951528549194, -0.8465834259986877, 0.036814432591199875, -1.2315269708633423, -1.5539002418518066, 0.1523015797138214, -1.1204482316970825, 1.4425984621047974, -0.6671543121337891, -0.19528645277023315, -0.22555981576442719, 1.7554042339324951, -1.2607853412628174, -0.23780788481235504, -0.5020462870597839, -0.3532162308692932, 0.48528334498405457, 0.03786429762840271, 2.447658061981201, -0.4314873516559601, 0.06128299608826637, 0.11891915649175644, -0.33434993028640747, -0.02381141297519207, 1.15432870388031, 0.9045704007148743, -1.1596602201461792, 1.0288100242614746, 0.8162977695465088, 0.23085856437683105, 1.3375900983810425, -1.6929012537002563, 2.0409095287323, 1.6173735857009888, 2.60231614112854, -0.24858850240707397, -0.11501822620630264, -2.234330177307129, 0.1436397135257721, 0.664290189743042, 1.2054569721221924, 1.1895986795425415, -0.6683743596076965, 2.1188395023345947, 1.5539839267730713, -2.0105013847351074, 0.12796607613563538, -0.6546440720558167, -0.1947394609451294, 1.180614948272705, 0.09675400704145432, 0.7005309462547302, -0.9921629428863525, -1.0740771293640137, -0.3925752341747284, -0.7731809020042419, -0.03553900495171547, -0.129952073097229, -0.6258066892623901, -0.5800729990005493, 0.1319647878408432, 0.22436416149139404, -1.2765657901763916, 0.7126895785331726, -0.8326931595802307, 0.8667321801185608, 0.28888416290283203, -1.582196831703186, 0.7686263918876648, -0.3085883855819702, -0.4320981800556183, -0.09030881524085999, -1.0235812664031982, 1.4535781145095825, 1.3808667659759521, 0.4403901696205139, 2.436093330383301, 1.3691965341567993, 1.7885581254959106, -0.2996942400932312, 0.8401139378547668, 1.401802897453308, -0.016085371375083923, 0.2321101576089859, 0.9849566221237183, -0.6912696361541748, -1.0648857355117798, 0.49694597721099854, -0.6341536641120911, -0.5491209030151367, 0.7166572213172913, -0.9755184650421143, 1.4087584018707275, -0.6510363221168518, 0.48675158619880676, -1.7557014226913452, -1.7459802627563477, -1.1528191566467285, 0.17045733332633972, 0.05956961587071419, -0.021277183666825294, -0.5609928369522095, -1.446179747581482, -0.21285946667194366, -1.4507255554199219, -1.328461766242981, -0.07508939504623413, -0.11533155292272568, 2.1652462482452393, -1.0951851606369019, -2.278545379638672, -0.44135117530822754, -0.7347334623336792, -0.025359492748975754, -1.439455509185791, 0.6366919875144958, 1.3453648090362549, 0.2400706708431244, -1.337549090385437, -0.2057461440563202, 1.097244381904602, 0.04413173347711563, 0.7572513818740845, -0.2086671143770218, 0.4054422378540039, -0.18059484660625458, -1.6853123903274536, 0.43525075912475586, 1.4342000484466553, 0.4985825717449188, 1.4429538249969482, 0.091854028403759, 0.19863902032375336, -0.30571112036705017, 1.1783928871154785, -1.2460243701934814, -1.2066118717193604, 0.8379859924316406, 0.7753110527992249, 1.3603025674819946, 0.09519916027784348, 0.9638319611549377, -1.0042083263397217, 0.3587782680988312, 1.1098908185958862, 0.9684984087944031, 0.43884220719337463, 0.6237095594406128, 1.9915175437927246, 0.7873385548591614, -1.476455569267273, 0.6960132122039795, -1.1839895248413086, 0.5789433717727661, 0.1395138055086136, 0.3205047845840454, -1.755301594734192, 0.696927011013031, 0.4359745979309082, 0.34485533833503723, 0.30181917548179626, 0.4637061059474945, 2.5520219802856445, 0.2679065763950348, -0.5103245377540588, -1.1639177799224854, -0.15253810584545135, 1.172757625579834, -0.6406981945037842, 1.428855538368225, 1.3508213758468628, -0.31926700472831726, 0.5815969705581665, 0.9662876725196838, 1.4579362869262695, 0.22005073726177216, 0.340771347284317, -0.23163476586341858, 1.3053091764450073, -0.4599345028400421, 1.0612151622772217, 0.46651092171669006, 1.3198457956314087, 1.6878079175949097, -0.1252661794424057, 1.2094136476516724, -1.6126844882965088, -0.018119556829333305, 0.526720404624939, 0.09498656541109085, 0.35947757959365845, 0.4523526728153229, -0.21022427082061768, -0.5722747445106506, -0.2944182753562927, -0.3930639624595642, -1.0237977504730225, -0.049690715968608856, -1.2385863065719604, 1.49948251247406, 0.12150293588638306, 0.022024547681212425, -0.8133159279823303, 1.5805870294570923, 1.2439804077148438, 1.7779252529144287, -0.3392632007598877, 1.7135484218597412, -1.1183745861053467, 0.0913306325674057, 2.003305673599243, 1.7233068943023682, -0.7697645425796509, 1.4468845129013062, -1.7809678316116333, -0.7922990918159485, 2.318005084991455, 0.04652927815914154, -0.37035229802131653, 0.9623851776123047, -1.5081348419189453, -0.04535311833024025, -1.607236623764038, -0.6069231629371643, 0.6023313999176025, -0.6466439366340637, 0.10369798541069031, 0.39372602105140686, 1.083367943763733, -0.9269680380821228, -0.25621089339256287, 0.4961181581020355, -1.654956579208374, -0.3468433916568756, -0.012829598970711231, -0.5458448529243469, -0.8551149368286133, 2.0959081649780273, 0.3634794056415558, -1.1294838190078735, 0.03467890992760658, 0.4938228130340576, 1.395743727684021, -1.4724642038345337, -0.21877719461917877, -0.6786037683486938, -0.7949038743972778, -1.071528434753418, 0.7949241995811462, -0.7050754427909851, 3.135002613067627, -1.027129888534546, -0.4554937183856964, 0.44681015610694885, -0.5831225514411926, 1.5693867206573486, 0.9980173110961914, 1.556032657623291, -1.7426189184188843, -0.40212351083755493, -1.3503508567810059, -1.3691023588180542, 0.19082455337047577, -2.0354630947113037, -0.6111645102500916, 0.8675597906112671, -0.03407956659793854, -0.9199312925338745, 0.30206069350242615, 0.39838048815727234, -0.1288941502571106, 0.8272150158882141, 0.12695413827896118, 1.2257370948791504, 0.48651254177093506, -0.46732017397880554, 0.13760735094547272, 0.3033446967601776, 2.1674962043762207, -0.545172393321991, 2.585761070251465, 1.0545251369476318, 1.3372740745544434, -1.348673939704895, -0.7138160467147827, 0.08199392259120941, -0.6077902317047119, -0.015022234059870243, -1.0309566259384155, 1.926836371421814, 0.2655131220817566, 0.005530090536922216, -2.4694039821624756, -0.30496746301651, -0.07636825740337372, -0.4904543161392212, -0.07628034800291061, 1.4432011842727661, -0.6289352774620056, 0.16711072623729706, 1.8559317588806152, -0.38048696517944336, 0.3612220883369446, 0.16210287809371948, 0.5846759676933289, 0.2139936089515686, 0.8658751249313354, 0.44622573256492615, 1.8287632465362549, 0.029413826763629913, -0.4944849908351898, -0.20701517164707184, -0.10194957256317139, -0.723511815071106, -1.3038606643676758, 0.904229998588562, 0.05126054584980011, 1.0049082040786743, -0.038366496562957764, 1.5350755453109741, -0.8099303245544434, 0.013015874661505222, 1.5003246068954468, 1.4945558309555054, 2.628466844558716, 0.08511143922805786, 0.04590126499533653, 1.5652685165405273, 0.6659771800041199, -0.792464554309845, 0.4722370505332947, 0.4199356138706207, 1.171181559562683, -0.22212587296962738, 0.6108567118644714, -0.9404696226119995, 0.7229167819023132, 1.423233151435852, 0.4164116680622101, 0.7600778341293335, -1.4396796226501465, 0.83985435962677, 0.45401397347450256, -0.9276853203773499, -0.0831526443362236, -0.7804126143455505, 1.3142554759979248, 1.470981240272522, -0.6693907976150513, 0.6171954274177551, -0.7073104381561279, -1.4115906953811646, 1.3479679822921753, 1.633049488067627, 0.5838363766670227, -0.2504742741584778, 1.2635173797607422, 1.1347293853759766, 0.8677460551261902, -0.025279266759753227, 1.972149133682251, -1.0351145267486572, -0.40348750352859497, -1.1592824459075928, 1.2788662910461426, 0.7064058184623718, 0.35562801361083984, 0.18094341456890106, 0.27788975834846497, -3.109764814376831, 1.3588619232177734, -0.20912909507751465, -0.7763366103172302, 0.11978194117546082, 0.1407966911792755, -0.004532643128186464, -0.5785140991210938, 1.1313915252685547, 0.13726672530174255, 1.2321010828018188, 0.0077211204916238785, 0.7934888005256653, 0.3052382171154022, -1.579716444015503, 0.8077570199966431, -0.06440741568803787, 0.040364351123571396, -1.289662480354309, 0.20366428792476654, 0.179717555642128, -0.6515231132507324, 0.967970609664917, 1.2657049894332886, -0.6836838126182556, -1.475131630897522, 1.0359610319137573, -0.7590054273605347, 0.3848145008087158, -1.0903658866882324, 1.2829718589782715, -1.0829529762268066, 0.7527757883071899, 1.3192510604858398, 1.2210646867752075, 0.9563241600990295, -0.24341008067131042, 2.1359686851501465, 0.4577684998512268, 0.021068785339593887, -0.08470644801855087, -2.1329023838043213, -1.4934816360473633, -1.2411319017410278, -0.09689214825630188, -0.42714112997055054, -0.35183894634246826, -1.395095705986023, 0.4713946282863617, -0.275164395570755, -0.4327269494533539, 1.0610849857330322, 0.939405620098114, 0.9702606797218323, -1.2673672437667847, -0.18523235619068146, 0.8424747586250305, -0.7388063669204712, 0.8933757543563843, -0.5890880823135376, 0.020908014848828316, -1.6541627645492554, 1.051526427268982, 0.23813803493976593, 0.7405998706817627, 0.4971086084842682, 1.0637513399124146, 0.009713510051369667, 0.10999801009893417, -0.16020238399505615, 0.6799406409263611, -0.5214394330978394, 0.7517301440238953, -0.8149849772453308, 0.31759902834892273, -0.16651594638824463, 1.2283679246902466, -0.48372551798820496, -1.9459590911865234, -0.7511064410209656, -1.358664870262146, 0.9679076671600342, 0.8851965069770813, 0.19371965527534485, 2.442697525024414, 0.19918809831142426, 0.08404736965894699, -0.5805211663246155, -0.15492364764213562, -1.400568962097168, -1.1894402503967285, 1.5904024839401245, -0.6154499650001526, 0.6505991816520691, -0.8718289136886597, 1.1198879480361938, -0.32819944620132446, -0.26999589800834656, 2.481278657913208, -0.2825758755207062, 0.7274687886238098, -0.15565165877342224, -0.3874475359916687, -0.7522674798965454, 0.3045133650302887, -2.899976968765259, 1.801992416381836, 1.9178050756454468, -1.059349775314331, -1.2415140867233276, 0.4996170103549957, -0.07182586193084717, 0.6138620376586914, -0.4180790185928345, -0.04567723721265793, 1.6606608629226685, 0.7179937362670898, -0.8693048357963562, 0.6552314758300781, -0.39128005504608154, -0.18749600648880005, 0.5924661755561829, 1.0548427104949951, -0.20114269852638245, -0.04042918235063553, -0.30068182945251465, 0.26535797119140625, 0.27956080436706543, 0.5468537211418152, -0.8058761954307556, -0.6366398930549622, 0.0639435350894928, -0.08669189363718033, -0.22375114262104034, -2.2459747791290283, 1.373518466949463, -0.8653645515441895, -1.865120768547058, 1.838637351989746, 0.780827522277832, 1.750128149986267, -1.449076533317566, -1.4221060276031494, -0.5081800818443298, 0.32489871978759766, -0.5724551677703857, -0.3666120171546936, 0.04662732407450676, -0.8052264451980591, -0.3459430932998657, -0.4538506269454956, -1.0740716457366943, 0.275250643491745, 0.03158557042479515, -0.36335569620132446, 1.1643409729003906, -0.04863893613219261, -1.6566452980041504, -0.10393970459699631, -0.002013683319091797, 0.4673701524734497, 0.8124045729637146, -0.7371432185173035, 0.04230988770723343, -0.6506314277648926, -1.7512685060501099, 1.6298928260803223, -0.24162761867046356, 1.2870817184448242, 0.948508083820343, -0.08301994204521179, 1.7609214782714844, -0.3200024664402008, -0.9858322143554688, -0.19342298805713654, 1.522577166557312, 0.7696930170059204, -0.050663817673921585, -0.3127886652946472, -0.7170432209968567, -0.3336271047592163, 0.28794169425964355, -0.04272126406431198, 0.7742982506752014, -1.261240005493164, -2.4248340129852295, 1.4281967878341675, 0.23545648157596588, -0.07685909420251846, 2.494704484939575, -0.8281297087669373, 0.9032915830612183, -2.314727306365967, -1.0738286972045898, -0.6633622050285339, -0.5860609412193298, 0.8612551689147949, -1.1682311296463013, -0.36997708678245544, -0.11894477158784866, -0.20727531611919403, 0.4293793737888336, -1.7037084102630615, -0.6670886874198914, 0.8151981234550476, -0.41250890493392944, -0.965704619884491, 1.0240304470062256, -0.8864285349845886, -0.7218568325042725, 0.8967639207839966, -0.6686629056930542, 1.460183024406433, 1.2407355308532715, 0.9795314073562622, -0.19296029210090637, 0.3068523705005646, 0.22888895869255066, -0.11654502898454666, -0.3966085910797119, 0.10265274345874786, -1.2912250757217407, 0.3147753179073334, -0.47149214148521423, -0.667742908000946, 0.27936434745788574, 1.326966404914856, -0.5680612921714783, -0.8648373484611511, -0.26474729180336, -0.9728787541389465, -0.17832419276237488, -0.026530297473073006, 0.3876015245914459, -0.3819938898086548, -0.4382424056529999, 0.7540658116340637, -0.6543170809745789, -0.33410391211509705, -0.10350613296031952, 0.7415757775306702, -0.0019259590189903975, -0.6867597103118896, 0.8330732583999634, 1.2684030532836914, -0.02829744666814804, 0.46080827713012695, -0.5037233233451843, -0.6901121735572815, -0.47790640592575073, 0.15340419113636017, -0.9345424175262451, -0.06736290454864502, -0.43241196870803833, -1.4290947914123535, 1.1790454387664795, -0.7603814601898193, 0.676961362361908, 1.0761430263519287, -0.6958603262901306, -0.9795427918434143, 0.4931407868862152, 0.40597841143608093, 0.7063623666763306, -1.0168825387954712, -1.0121079683303833, 0.14744171500205994, -0.610255241394043, -0.666038453578949, 0.40252685546875, -1.1323543787002563, -0.6233253479003906, 1.4922900199890137, 0.9954364895820618, 1.0543689727783203, -0.19964662194252014, -0.19685551524162292, 0.20801176130771637, 0.6458064913749695, -0.4034830927848816, -1.3923419713974, -1.2189879417419434, -1.6554549932479858, 0.1957714855670929, -0.04620283842086792, -0.3621633052825928, -0.7812222242355347, -0.32623225450515747, 1.0879744291305542, 0.12659074366092682, 1.1542272567749023, -0.13822102546691895, 0.6192190647125244, 1.0989121198654175, 2.4832701683044434, -1.6762422323226929, -0.3164641261100769, -1.0181796550750732, 0.6596629023551941, -0.823957085609436, -1.5378644466400146, 0.16482724249362946, 0.7697803974151611, -1.3469948768615723, -0.10076767206192017, -1.5842021703720093, 0.9451332688331604, -0.23487356305122375, -0.37787237763404846, 0.2778087258338928, -0.07111147046089172, 0.4514959156513214, 1.251853585243225, -1.6857582330703735, -1.4103854894638062, 1.5982184410095215, -0.8907670974731445, -1.5319608449935913, -1.385648488998413, -0.5770448446273804, 1.2336047887802124, -0.13959495723247528, -1.0362578630447388, 0.07433093339204788, -1.6897985935211182, 0.9351157546043396, -0.825718104839325, -0.727419912815094, 0.08985079824924469, -1.372185230255127, -1.343729853630066, 1.0437601804733276, -0.5223090648651123, 0.7529641389846802, -1.5851328372955322, 1.2123088836669922, -1.4111814498901367, -1.1972095966339111, 0.3636684715747833, -1.1539642810821533, -0.6366005539894104, 1.8489631414413452, -0.8781163692474365, 1.0101815462112427, 0.8032150864601135, 0.16001763939857483, 0.3613402247428894, -0.5751116871833801, 1.702561378479004, 0.8919855952262878, 0.32943227887153625, 1.113531231880188, 0.7939905524253845, 0.2001442313194275, 0.5341563820838928, 0.06124381721019745, 0.27713319659233093, 0.9896803498268127, -0.8736057281494141, 1.018169641494751, 0.6209961771965027, -1.205161213874817, -0.8647321462631226, -0.1100400760769844, 0.7878108620643616, 2.2190005779266357, 0.9821866154670715, -0.6898482441902161, 0.7681911587715149, 0.05323782190680504, -1.4746742248535156, -0.7356196641921997, -0.36327067017555237, 0.708306074142456, -0.9154846668243408, 1.1413593292236328, -0.7127847075462341, 1.1094069480895996, -2.0457630157470703, -0.7055767178535461, 0.2964142858982086, 1.9796748161315918, -0.007777743507176638], [0.7978259325027466, 0.2022525668144226, 0.06569413840770721, 1.112611174583435, 0.054106079041957855, -0.702712893486023, 0.35915374755859375, 1.2292475700378418, -0.5993122458457947, 0.27403005957603455, -1.3527560234069824, -1.050430417060852, 1.243177056312561, 0.9179856777191162, 0.012339246459305286, 0.9375951290130615, -2.0235788822174072, 2.0368812084198, -2.0925304889678955, 0.18002408742904663, -0.26360076665878296, 0.7375405430793762, -0.17934022843837738, -0.5627384185791016, 1.3144584894180298, 0.23072800040245056, -1.6963614225387573, -0.41515475511550903, 1.0292774438858032, 0.43238651752471924, -0.5727763772010803, 0.11219529807567596, -0.08325748890638351, -0.565269410610199, -1.8014404773712158, 0.7445712685585022, 0.20613659918308258, -0.9748836159706116, -0.1304880529642105, -0.19439174234867096, -0.9228909611701965, -0.584536075592041, -0.7686304450035095, -0.4730529487133026, -2.234335422515869, -0.11945725977420807, 0.17143496870994568, -0.446714848279953, -1.728845238685608, -0.1701509952545166, -0.3502779006958008, 2.118300676345825, 0.4220142662525177, 0.4732135832309723, -0.5556685924530029, -0.4229450225830078, -0.36426207423210144, -0.2559722661972046, 1.0607085227966309, 0.22586803138256073, 0.11050144582986832, 1.6974660158157349, 1.0284491777420044, 1.1250014305114746, 0.40880584716796875, -1.3062008619308472, 0.26312097907066345, 0.06511780619621277, -0.2763022184371948, 0.5123820900917053, 0.05280524864792824, -1.5176738500595093, -1.1254078149795532, -0.2973448932170868, 1.131330132484436, -0.20889706909656525, 1.0905009508132935, 0.3476138710975647, 0.7771439552307129, -1.8085795640945435, -1.097184181213379, 0.4259273409843445, -0.9567150473594666, -0.9723612666130066, -0.4274477958679199, -2.0989489555358887, 1.2370061874389648, -1.3202977180480957, -1.5488864183425903, -0.4661961793899536, 1.884002685546875, -1.5291037559509277, 0.5986742973327637, -0.005410111043602228, 1.2409330606460571, -0.194421648979187, 0.3036311864852905, -0.21080252528190613, 0.5573191046714783, -1.7059986591339111, -0.34011536836624146, 0.17715242505073547, -0.7946369051933289, 0.24010029435157776, -0.6965821385383606, -0.37766599655151367, -0.4338024854660034, -0.8528203964233398, 0.6469270586967468, 1.7035670280456543, -1.6283656358718872, -2.5040371417999268, 0.459197461605072, 0.1723204404115677, 0.36880484223365784, 0.11264293640851974, -0.21397855877876282, 0.23381361365318298, -0.9007838368415833, 0.6189083456993103, 1.1557897329330444, 0.37271103262901306, 0.634052574634552, 0.5967046618461609, 0.4104751646518707, 0.8869621157646179, -0.16845181584358215, -0.12275966256856918, -0.38830500841140747, 0.04253261536359787, 0.07643122225999832, -1.4201045036315918, 0.9284229278564453, -1.3708992004394531, -2.438410520553589, -1.8375539779663086, -0.49364399909973145, -0.31617867946624756, -0.6106278300285339, -0.7286413311958313, 1.0638991594314575, -0.7712746262550354, 1.6030148267745972, -0.6385716199874878, 0.6818477511405945, -1.451069712638855, -0.7243261337280273, -0.2786496579647064, 1.4802336692810059, 0.14006757736206055, -0.10132808238267899, -1.2344263792037964, -0.7101635932922363, 0.811572790145874, 1.235547423362732, -0.07970177382230759, 0.14144271612167358, 1.194724202156067, -0.5560504794120789, 0.28380998969078064, -0.6909328699111938, 1.8206405639648438, -0.7646229863166809, 0.18823418021202087, 0.6157540678977966, -0.7059299945831299, -1.9407966136932373, 0.0719594806432724, -0.23628367483615875, 2.4983112812042236, -0.38425785303115845, -1.8845388889312744, -0.7454537749290466, 1.0681052207946777, -0.8666635751724243, -0.5719345211982727, -1.736635446548462, 1.1228840351104736, -1.0212886333465576, -0.9361225962638855, 0.32511699199676514, -0.502815306186676, -0.2913713753223419, -1.363571047782898, 0.7746546864509583, 1.406731128692627, -0.12711907923221588, -0.0027082813903689384, -0.5250557661056519, -0.46434125304222107, 0.6524433493614197, 0.996887743473053, -0.29789066314697266, -0.41610535979270935, 0.37883615493774414, 0.2759638726711273, 1.93925940990448, 2.028367757797241, -0.13564950227737427, -0.07667659968137741, -0.1689685434103012, 0.4884170591831207, -0.5524197220802307, 1.3711395263671875, 0.6069010496139526, -1.7632887363433838, -0.5864890217781067, -1.321517825126648, 1.1771246194839478, 1.9150696992874146, 0.9026874303817749, 0.11225505918264389, 1.8610979318618774, -0.19224397838115692, 0.21665264666080475, -0.5961097478866577, -0.6147007346153259, 0.10835369676351547, 0.1379227340221405, -0.2849195897579193, -0.1311943680047989, 1.5290085077285767, -0.5569565892219543, 0.6166380047798157, 0.7112796902656555, -0.8605706691741943, -1.3617953062057495, -0.04314751923084259, 0.7037724852561951, 0.30247658491134644, 0.2782180607318878, 0.2646894156932831, 0.36358562111854553, 0.21631309390068054, -0.47803494334220886, -0.2904669940471649, -0.08408398181200027, -0.22931919991970062, -0.33819931745529175, -1.223563551902771, -0.6855875849723816, -0.8788039684295654, 0.15149392187595367, -0.8353561758995056, -0.3753437399864197, -1.2946749925613403, -0.8671800494194031, -0.3820720613002777, -1.681455373764038, 1.6906577348709106, -1.3354315757751465, -1.1829882860183716, -0.6761654019355774, 0.4501638412475586, -0.35842880606651306, 1.8518067598342896, -0.32980313897132874, 0.39475637674331665, -0.7709835767745972, 0.597327709197998, 0.9008798599243164, 1.5253480672836304, -0.6713454723358154, 0.7555874586105347, -0.3569982647895813, 0.671385645866394, 1.7220326662063599, 0.22371526062488556, -0.08585488051176071, -0.7845005989074707, 0.4253247082233429, 0.8729495406150818, -1.4682154655456543, -0.6720475554466248, 1.1766464710235596, -0.3171824812889099, 0.9690358638763428, -1.7598158121109009, -0.04193880781531334, -0.4176504611968994, -0.505064070224762, -0.6292367577552795, -0.1595744788646698, 0.12250976264476776, 1.3274033069610596, -1.3030661344528198, -0.7248125672340393, 0.37303486466407776, 1.097756266593933, 0.9903196096420288, 0.7305120229721069, -0.9300636649131775, -1.179543137550354, 1.8926533460617065, 0.48554712533950806, -1.4333425760269165, 0.08551899343729019, 0.11777583509683609, 1.2640036344528198, 0.2487456202507019, -0.5444015264511108, 2.1838200092315674, -0.7641606330871582, -0.3624967932701111, -0.6391969323158264, 0.5388749837875366, 1.4585621356964111, -0.8951523900032043, 0.08817458897829056, -1.858907699584961, 0.4569838047027588, 1.0778034925460815, -0.47312813997268677, 0.19449084997177124, 2.070977210998535, 1.1147481203079224, 0.3304097354412079, 0.36007601022720337, 0.3339771628379822, 0.3265415132045746, 0.3963811993598938, -0.17420770227909088, 1.4417054653167725, -0.6825132369995117, -1.1865606307983398, 1.1626994609832764, -1.894950032234192, 1.7927896976470947, -0.4710577428340912, -1.3620671033859253, 1.375214695930481, -1.8622701168060303, 1.3072667121887207, -0.1811715066432953, -0.2720395624637604, -0.5481609106063843, -3.086634874343872, -0.14058008790016174, -0.23751503229141235, -1.820504069328308, 0.45760902762413025, 0.7778500914573669, 0.37565290927886963, 0.1377965211868286, -0.06203101947903633, -0.23970510065555573, -0.09065119177103043, 0.9605396389961243, 2.0019893646240234, 2.145879030227661, 1.5041871070861816, 0.7013431191444397, 1.3532942533493042, 1.0604733228683472, 1.8039242029190063, -1.0407190322875977, -1.3594615459442139, 0.40242958068847656, 0.8281311988830566, 0.1883770078420639, -1.1206172704696655, -0.5124502778053284, -0.13898690044879913, -0.41118794679641724, 0.36614224314689636, -0.6376215815544128, 1.1149213314056396, -1.164452075958252, -0.23300854861736298, -1.338240385055542, 1.1217942237854004, 0.18463435769081116, -0.3250432312488556, -1.9970899820327759, 1.710590124130249, -0.5179097056388855, -0.2742230296134949, -0.5778385996818542, -0.3073168098926544, -0.4165685176849365, -0.13441765308380127, 0.17556382715702057, 0.7850453853607178, 1.3293131589889526, -0.7003596425056458, -0.406353622674942, 0.19950903952121735, -1.9024744033813477, 0.4337690770626068, -0.8121243119239807, -0.31239551305770874, -1.5506176948547363, -0.20043836534023285, 0.004430318716913462, 0.04910328984260559, 2.579741954803467, -0.4921857714653015, -2.0983381271362305, -1.4040447473526, -0.4266623258590698, -1.3813858032226562, -1.1309787034988403, 0.3957614004611969, -0.23668105900287628, -0.8507712483406067, 0.4649810791015625, -1.3257938623428345, -0.4831372797489166, -0.9371243715286255, -1.0704572200775146, 0.7788773775100708, 1.8058140277862549, -0.35368993878364563, 2.2087035179138184, -1.90848708152771, -0.31787094473838806, 0.19541427493095398, 1.864802598953247, 0.7451109886169434, 0.06697406619787216, -0.22423823177814484, 0.7511969208717346, -1.6052335500717163, -0.629939615726471, -0.5348829627037048, 0.21594564616680145, 0.747567355632782, 0.45676666498184204, -0.5846465826034546, 0.4587460458278656, 0.8515316247940063, -0.2126283049583435, 1.7310292720794678, 0.0670585110783577, 0.8748626708984375, 1.7127206325531006, -0.12024145573377609, -0.29478099942207336, 0.5063568949699402, 0.66668301820755, -2.436962842941284, 0.2395707070827484, -1.4062031507492065, 0.20194481313228607, 1.8733845949172974, 0.7615401744842529, 0.9398521184921265, -0.7396456599235535, 0.7975285649299622, 0.7625037431716919, 0.6267619729042053, -0.2532953917980194, -0.19791874289512634, -0.5607894659042358, -0.18587422370910645, 0.9972789883613586, 0.16223256289958954, -0.8063855767250061, -0.34422817826271057, -0.8720853924751282, -0.00349027244374156, -0.8833584785461426, -0.8361856937408447, -0.26987695693969727, -0.8845812082290649, -0.5885500907897949, -1.1177219152450562, 0.515955924987793, -0.8232972025871277, 0.29087311029434204, 0.37705564498901367, -0.2698519229888916, -0.0425371490418911, -0.3872692286968231, 0.09827044606208801, -0.6890613436698914, 0.14803743362426758, -1.1070518493652344, 1.5392094850540161, -1.9343818426132202, 0.9963248372077942, -1.6268954277038574, 0.06905026733875275, 1.2273457050323486, 0.3268235921859741, -0.43440520763397217, -0.6143434047698975, -0.6602838039398193, -0.09143570810556412, 0.5737915635108948, -1.229764699935913, 0.23248597979545593, 0.30374035239219666, -0.5873668193817139, 1.4470202922821045, 0.17584186792373657, -0.14525887370109558, 1.1209831237792969, 1.7996199131011963, -0.1268189698457718, 1.4418801069259644, 0.01989920809864998, -0.7415069937705994, 0.019076736643910408, -0.3647068738937378, -0.13693568110466003, 2.6803183555603027, 0.1577121615409851, 0.3802517354488373, -0.002734886249527335, -0.6572022438049316, 0.06793410331010818, 0.37817978858947754, 0.50706946849823, -1.3028113842010498, 0.1539193093776703, 0.7396160960197449, -0.05396323651075363, -0.02299182489514351, -1.8905915021896362, 0.10256706923246384, 0.5350620150566101, -0.5716584920883179, 0.781914472579956, 0.4760369658470154, -0.3827208876609802, 0.1407942920923233, 1.3314130306243896, -0.6329154372215271, -0.28516167402267456, 0.36908721923828125, 0.08045226335525513, 0.7658271193504333, -0.1469273567199707, -1.745343565940857, -1.4602409601211548, -0.2878345847129822, 0.28329527378082275, 1.094970941543579, 0.06970403343439102, -0.2114500254392624, -0.7340075969696045, -1.2037324905395508, 0.9272105097770691, 1.2152187824249268, 1.081695556640625, -0.8318343162536621, 0.8538889288902283, -0.4214920699596405, -0.3727209270000458, -1.3422774076461792, -0.43045973777770996, -1.922830581665039, -1.1055881977081299, -0.4997251331806183, -2.542376756668091, -0.14997662603855133, -0.44261181354522705, -0.946299135684967, 0.47866183519363403, -1.117708444595337, 0.020270144566893578, -0.24510370194911957, 0.04466995969414711, -2.206468343734741, 0.7919126749038696, 0.25601378083229065, -0.2586684823036194, 0.474619060754776, -1.594926357269287, 0.21105623245239258, 0.3727804124355316, -0.18173572421073914, 1.156835913658142, -0.27415311336517334, 0.8933084011077881, -1.380376935005188, 0.3554123342037201, 1.0021061897277832, -0.19027085602283478, -0.579986035823822, -0.07129251956939697, -0.24507516622543335, 0.13999421894550323, -0.7832708954811096, 0.20219801366329193, -0.303148478269577, 1.4935320615768433, -2.358248710632324, -0.7644001841545105, 1.1738333702087402, -0.8956311345100403, 0.9365630149841309, -0.2519873380661011, -0.07577389478683472, 1.3789536952972412, -0.5326172709465027, 0.2208089977502823, -1.8262228965759277, -1.0819363594055176, -1.0013067722320557, -0.0244152694940567, 2.1726534366607666, 0.10433387756347656, 0.686639130115509, 0.13167376816272736, -0.13877087831497192, 0.08893091231584549, -1.1598986387252808, -0.07646923512220383, 0.6756303310394287, 1.7229442596435547, 0.7935940027236938, -0.6682305335998535, 0.16503211855888367, 0.06937313079833984, -0.6629126667976379, -0.18766357004642487, -0.8892219066619873, 0.4508701264858246, -2.072695732116699, 0.956122875213623, 1.7663580179214478, 1.370795488357544, 0.7694160342216492, 0.6297783851623535, -0.16919679939746857, -0.30953356623649597, 0.5307801961898804, 0.868973970413208, -0.4167860448360443, -0.800504744052887, -1.2148590087890625, 2.0026187896728516, -0.24178244173526764, 0.878110945224762, 0.2727048993110657, 0.9654836058616638, -0.8248940706253052, 0.42759692668914795, 0.6201409697532654, -0.716310441493988, 1.141338586807251, 0.12963959574699402, 1.486798644065857, 0.4695760905742645, 1.9001107215881348, -0.012117656879127026, 0.9427405595779419, 0.29432475566864014, -0.9145791530609131, 1.4078203439712524, 1.0439643859863281, 1.659123420715332, 0.8917943835258484, 0.8150864839553833, -0.5316652059555054, 0.5214639902114868, 1.2330113649368286, 0.3377690613269806, 0.7294141054153442, 0.10067327320575714, 0.8788171410560608, -0.07682778686285019, 0.48590531945228577, 0.22589024901390076, -0.041503019630908966, -0.15962667763233185, -0.30976682901382446, 1.3357183933258057, 0.4884818494319916, -0.9457268714904785, -1.3594152927398682, 0.01613214798271656, 0.1652199923992157, 1.681776523590088, 0.05986982211470604, 1.4491069316864014, -0.11797440052032471, -0.5845253467559814, 0.3721713125705719, 0.5061331987380981, -1.0132534503936768, 1.1510499715805054, 1.1280083656311035, -0.7351380586624146, -0.42214807868003845, 0.5771621465682983, -0.500281810760498, 0.2986201047897339, -0.8223903179168701, -0.2347441166639328, 0.42436331510543823, -0.011377330869436264, 0.36401185393333435, 0.9521358013153076, -0.48217257857322693, 0.2876209020614624, -0.9517732858657837, 2.606138229370117, 0.19111062586307526, -2.1567494869232178, 0.7093269228935242, -0.2786116302013397, 0.8823522925376892, -0.6244503855705261, -1.2434968948364258, -0.04335331916809082, -1.1380285024642944, -0.8173715472221375, 0.16201983392238617, -0.6053673624992371, 0.792795717716217, -0.567659318447113, -0.5741217136383057, 1.3283586502075195, 1.1896753311157227, 0.5902321934700012, 1.5387499332427979, -1.5427618026733398, 0.8177289962768555, 0.6610174179077148, -0.48173627257347107, -0.49436110258102417, 0.14596189558506012, -0.817201554775238, 0.6852344274520874, 0.3213956654071808, 0.842795729637146, -0.8683071732521057, -0.8179923892021179, -0.9825871586799622, 0.8543829321861267, -0.44529715180397034, -0.9931830763816833, 1.5204980373382568, -0.6649927496910095, -0.7864344716072083, -0.4553860127925873, 0.30935779213905334, 0.05149196833372116, 0.008908436633646488, 0.1278538703918457, -1.5124244689941406, 0.06823578476905823, 1.843645691871643, 0.7215623259544373, -0.38345277309417725, -0.007685328368097544, -0.8212692737579346, -0.5551977753639221, 1.0276572704315186, 0.2706682085990906, -0.8415890336036682, 1.5218242406845093, 1.5442534685134888, 0.17431727051734924, 1.3313896656036377, -1.1751893758773804, -0.14864709973335266, -0.31369781494140625, 0.5292180776596069, 1.4378414154052734, -0.4399624764919281, -0.15693700313568115, 1.4592145681381226, 1.463705062866211, 0.7227442860603333, 0.9917783737182617, 1.0761674642562866, 0.44318732619285583, 1.422059178352356, -1.2539297342300415, 0.4372420608997345, 0.9404578804969788, 0.15731172263622284, 0.28672799468040466, -1.0908339023590088, 0.8568829298019409, -1.1295393705368042, 0.4627409875392914, 1.4127377271652222, -2.2686328887939453, 0.2983577251434326, 2.2675514221191406, 0.14088258147239685, 1.06714928150177, 0.49603140354156494, -0.12814132869243622, 1.4970623254776, 1.4230891466140747, 2.1879756450653076, -0.6374529004096985, -0.6505181789398193, 0.18656502664089203, -0.08520720899105072, 0.8535630702972412, -1.491626501083374, -0.11001261323690414, 0.8687321543693542, -0.8304187655448914, -0.3038944900035858, -0.1448875069618225, -0.18989914655685425, -0.5777705311775208, -1.2717194557189941, -0.967746913433075, -0.23728440701961517, 1.5361095666885376, 0.8074966669082642, -0.870821475982666, 1.2341415882110596, -0.3984617590904236, 0.7550641298294067, 1.3632203340530396, 0.8052179217338562, -0.05421047657728195, 0.3857446014881134, 1.2195024490356445, -1.4074602127075195, 0.7305634021759033, 0.7626433968544006, 2.168278455734253, -0.844620943069458, -0.8596800565719604, 0.9683211445808411, 0.3314327597618103, -0.7205868363380432, 1.322818636894226, 0.3888419270515442, -1.7431020736694336, 1.1160310506820679, -1.971419095993042, -1.2560760974884033, -0.5522996187210083, -0.7654605507850647, -1.0357102155685425, 1.7199766635894775, 1.3287376165390015, -1.8562535047531128, -0.0115638617426157, 0.24047334492206573, 1.1221923828125, -0.7374016642570496, 1.2807561159133911, 0.4627856910228729, 0.8690129518508911, -1.61057710647583, 1.2987290620803833, 0.4426659643650055, -0.4675391614437103, 2.1092607975006104, 0.9394831657409668, 0.10927747935056686, 0.5531909465789795, -0.6234618425369263, 0.09340082854032516, -1.565439224243164, -0.41761502623558044, -0.2020360380411148, -1.3741358518600464, 0.6584944725036621, -0.6815029382705688, -1.6679015159606934, 1.7478450536727905, -0.5067259073257446, -1.5630009174346924, 2.2297146320343018, 0.6175269484519958, -0.7759718298912048, -0.13075929880142212, -1.5268398523330688, 0.8396958708763123, 0.6241176724433899, -0.42952343821525574, 0.6099418997764587, 1.517751693725586, 0.3671649992465973, -1.413745641708374, 0.37778589129447937, -0.87688809633255, 0.1177411749958992, -0.16419482231140137, -1.5056874752044678, -0.5862640738487244, 0.48623862862586975, -0.3756754994392395, 0.8477447628974915, -1.0027812719345093, -0.4495810568332672, -0.808295726776123, -1.0399754047393799, 1.7762806415557861, 0.653596818447113, -1.0405350923538208, -1.183977484703064, 0.6125147342681885, 0.12852834165096283, -0.09324049949645996, 1.5981322526931763, -0.06383378058671951, 1.0332313776016235, -0.28438761830329895, 0.9814537763595581, -0.2751811444759369, -1.6307975053787231, -0.613896906375885, 0.7648119330406189, -1.1836177110671997, 1.1351128816604614, 1.2693625688552856, 0.236343115568161, -1.1735601425170898, -1.7266803979873657, -1.6783428192138672, -0.0683884471654892, 1.0693528652191162, 1.238680124282837, -1.211874008178711, -0.06351545453071594, -1.441176176071167, -0.6210132837295532, -0.15646928548812866, 0.11299045383930206, -0.9938521981239319, 0.8146231770515442, -0.14736640453338623, -0.5497138500213623, -0.48495733737945557, 0.21396097540855408, -0.8391898274421692, 1.0224120616912842, 0.620391845703125, -0.9045317769050598, -0.26245301961898804, 0.5668156743049622, 0.37303754687309265, -0.41238242387771606, 0.6246726512908936, 0.8863155841827393, -0.5349653959274292, 0.5838636755943298, -0.9063923954963684, -1.0035011768341064, -0.33105480670928955, -1.0077232122421265, 0.37147629261016846, -1.011836290359497, -1.0839184522628784, -1.3455489873886108, -0.9126265645027161, 0.8086011409759521, 0.6047115921974182, 0.8218625783920288, 0.40194395184516907, -1.7573472261428833, 2.1082708835601807, 0.7258250713348389, 0.8214012384414673, 0.21491092443466187, 1.4154554605484009, 0.8134044408798218, 0.5808439254760742, -1.7455682754516602, 0.6008720397949219, 1.084358811378479, -0.549762487411499, -0.7688965201377869, 1.6791571378707886, 0.693770170211792, -0.08168762177228928, -0.15996627509593964, -0.6972259283065796, 0.2152802050113678, -0.5002919435501099, 0.05905567854642868, 1.4056628942489624, -1.341719627380371, -1.1103631258010864, 0.8866667151451111, -1.4815828800201416, 1.0424706935882568, -0.9207333922386169, 0.5408265590667725, -0.8117060661315918, -0.07207674533128738, 1.524015188217163, -1.3693147897720337, 1.5142515897750854, -0.8378084897994995, -0.31819579005241394, -1.0484223365783691, -0.5775793194770813, 0.41394272446632385, 1.1121519804000854, 0.9586530923843384, -0.25036895275115967, 0.9096773266792297, -0.12311471998691559, -0.1725589483976364], [-0.7231809496879578, 1.026079535484314, -0.07968506962060928, 0.08877478539943695, -0.5635979771614075, -1.1084462404251099, 0.9093915820121765, -0.3294762074947357, 0.8499363660812378, -0.327820360660553, 0.3192227780818939, 0.22985467314720154, 0.407591313123703, -1.605645775794983, 0.12684212625026703, -0.24694593250751495, 1.9188984632492065, 2.1437318325042725, -1.5850181579589844, 1.561080813407898, 0.5750486850738525, 0.7771872878074646, -0.04705379903316498, -0.8027649521827698, -0.6750057935714722, 0.08187320828437805, 0.3760705292224884, 2.3420088291168213, 2.1956918239593506, -0.2425379902124405, -0.3809305429458618, 0.10236058384180069, 0.6245046257972717, -0.49210503697395325, -0.31074824929237366, 0.8409337401390076, -0.3161699175834656, -0.13125070929527283, 2.053696632385254, 0.6003475189208984, -0.6768861413002014, -2.0169644355773926, -1.8110578060150146, -0.021165449172258377, -0.3743993639945984, -1.6848982572555542, 0.6359757781028748, -0.5129626989364624, 1.0620039701461792, 0.6234392523765564, -0.14681780338287354, 0.3191003203392029, 0.03282951936125755, -2.5620357990264893, -1.225758671760559, -0.8159421682357788, -0.14875172078609467, -1.598436951637268, -0.6430796384811401, -0.2921794056892395, -0.2374316155910492, 0.27570945024490356, 0.19022896885871887, 0.11839095503091812, -1.0300244092941284, 1.2701516151428223, -0.16138094663619995, 1.2162286043167114, 0.4387175440788269, 0.948171854019165, 0.8346523642539978, -1.3286094665527344, -0.2612041234970093, -0.4717423617839813, 0.9682389497756958, -0.7662822604179382, 0.18138891458511353, 0.07376766204833984, -1.0919063091278076, 1.8330835103988647, 2.1879677772521973, -0.20697003602981567, 0.2947829067707062, -0.8829336762428284, -0.9392714500427246, -0.20066967606544495, 0.9464948773384094, 0.21156327426433563, 0.01757522113621235, -0.7421132326126099, -0.49977120757102966, -1.0105150938034058, -0.7263440489768982, -1.571678876876831, 0.04923751950263977, -0.6304576396942139, 1.9225512742996216, -0.5612966418266296, -1.8196988105773926, -0.09817268699407578, -0.05077477544546127, -0.4970073401927948, 1.563254952430725, 1.5307190418243408, 0.0238228477537632, 0.5823951363563538, 0.4663253426551819, 0.2762685716152191, 0.3834732174873352, 0.8084303140640259, 0.8494418859481812, -1.4050956964492798, -0.004786859266459942, 0.8019247055053711, -1.3699268102645874, -0.20969192683696747, -1.1544793844223022, 0.4924607276916504, 0.2663860321044922, 1.5757148265838623, 1.2068928480148315, 0.6758683919906616, -0.5177876949310303, 0.8618329167366028, 2.5604753494262695, 0.7606736421585083, 1.1894252300262451, 1.483595371246338, -1.636419415473938, -0.5221421718597412, 0.5141590237617493, 0.7861311435699463, -1.136851191520691, 0.8012204766273499, -1.1773320436477661, -0.07278946042060852, 0.157033309340477, 0.38197407126426697, -1.4366883039474487, -0.19840772449970245, -0.8483220934867859, 0.035194236785173416, 1.201965093612671, 0.39732980728149414, -1.435897946357727, 0.6487510204315186, 1.104901671409607, -0.13734525442123413, 1.0127743482589722, 1.7871153354644775, 0.025309301912784576, -0.37691909074783325, -0.40022012591362, -1.5384330749511719, -1.045639991760254, 0.7742581963539124, 0.8603612780570984, 1.1382126808166504, -0.6974053382873535, -1.7750869989395142, -0.8124062418937683, 0.4642725884914398, 1.2122175693511963, -0.6863162517547607, 1.3116813898086548, 0.49117255210876465, -0.554797351360321, 0.6596857905387878, 0.835425615310669, 1.1843193769454956, -0.9879304766654968, 0.5408983826637268, 0.7194938063621521, 0.5961829423904419, 0.3488362431526184, 0.7334139943122864, 0.06468193978071213, 0.23635327816009521, 2.102090358734131, 0.4633183479309082, -0.2783636152744293, -0.9718208909034729, 0.26919442415237427, 0.144414022564888, 0.5319497585296631, 1.3216031789779663, -0.9771755337715149, -0.6453438401222229, 1.0513255596160889, -1.2195748090744019, 0.7876902222633362, -1.077209711074829, -0.5935945510864258, 0.6977183818817139, -0.25294023752212524, 1.318015694618225, -1.0064979791641235, 2.2050461769104004, -0.18696168065071106, -0.30511751770973206, 0.1393711119890213, 0.33172786235809326, -0.14851468801498413, 0.5052763819694519, 0.07838978618383408, 0.09966208040714264, -0.5226045250892639, -0.873555064201355, -1.3522121906280518, -0.5518113970756531, 0.9095801711082458, 0.7761009335517883, 0.39224565029144287, -1.1014046669006348, 0.6206751465797424, -1.9072190523147583, 0.8535737991333008, 0.5105027556419373, -0.19431006908416748, -1.1827824115753174, 1.183782696723938, 2.4044697284698486, 1.22938871383667, 1.741092562675476, -0.43054473400115967, -1.71999192237854, -1.4904941320419312, 0.5532395839691162, 0.41183438897132874, -2.1609911918640137, 0.0914752185344696, -0.3154718279838562, -0.22081297636032104, -0.8421425819396973, 1.3238102197647095, -1.6364408731460571, -0.1825755089521408, -1.2444173097610474, -0.7412915229797363, -0.01593843661248684, 0.6685876846313477, -0.14948345720767975, -0.11400751769542694, 0.7223038673400879, -0.9275656938552856, 0.39074474573135376, -1.5212730169296265, 0.3123110234737396, 0.16293613612651825, 0.3431117534637451, 0.30531302094459534, 0.6848624348640442, -0.5661587119102478, 0.16659586131572723, -0.9116576313972473, -0.028398888185620308, -0.21845021843910217, 1.0216021537780762, -0.20224231481552124, 0.1810133010149002, 0.3163481056690216, -0.7907030582427979, -1.4046810865402222, 0.012307289056479931, -2.448164224624634, -1.002310872077942, 1.2878937721252441, 1.4579755067825317, -0.6138896942138672, 0.5098899602890015, 0.007603210397064686, -1.9223366975784302, -0.1667378544807434, -0.6209842562675476, -0.1820053905248642, 1.4307101964950562, 0.8138113021850586, -1.397286057472229, -0.09908656030893326, 0.5125207901000977, 0.23675204813480377, 0.38662293553352356, -1.8432502746582031, 0.9630150198936462, -1.219132900238037, 1.415283441543579, 0.01555967889726162, 0.5896020531654358, -0.17387422919273376, 0.3840502202510834, 0.03771420195698738, -1.7380932569503784, -1.1653438806533813, -0.8824580907821655, 0.5357429385185242, -0.06937014311552048, -0.3043021261692047, 1.098541021347046, -0.30421391129493713, -0.7631941437721252, 0.536833643913269, 0.5126698017120361, 0.6420650482177734, 1.8247319459915161, 0.25657251477241516, -0.994805097579956, -0.5649827122688293, 1.0416371822357178, -0.9246209263801575, 0.527646005153656, 0.895665168762207, 0.02643791027367115, -2.230916976928711, 0.3582894206047058, -0.7585518956184387, 3.465615749359131, -0.42112183570861816, 0.8606472611427307, -0.3689209222793579, 1.3084439039230347, 0.2740110456943512, -0.6770244240760803, 0.670074999332428, 0.15944179892539978, 0.17156101763248444, -0.8977858424186707, 1.7725224494934082, -0.4982321262359619, -0.3164002299308777, 0.8182952404022217, 0.8254474401473999, -1.0863018035888672, -1.0559828281402588, 0.9610813856124878, -1.2312887907028198, -1.7033884525299072, 0.6902183294296265, -0.201684832572937, -0.8249848484992981, -0.7558556199073792, 0.025932947173714638, -0.3441755175590515, -0.08963099867105484, -0.08383394777774811, -0.5499971508979797, -0.8240388035774231, -0.39612171053886414, 0.5560506582260132, -0.0471358485519886, 1.2570717334747314, -1.156796932220459, 1.226088285446167, 1.656819462776184, 0.30864009261131287, 1.072514295578003, 1.4251155853271484, -0.3340994417667389, 0.08143945783376694, -2.1808223724365234, 0.6873084306716919, -0.39510801434516907, 0.9409541487693787, 0.758975625038147, -0.8229302167892456, 0.6382776498794556, -0.4949789345264435, 1.0339609384536743, -0.17143838107585907, 0.879652202129364, 0.6208221912384033, 0.03328467905521393, 0.048874907195568085, -0.5383833050727844, -0.16150687634944916, -0.027160512283444405, -0.6219968795776367, -1.4593865871429443, -0.9284791946411133, 0.47189605236053467, 1.0358942747116089, -0.2889537513256073, -0.6078843474388123, 0.15003696084022522, 0.6202203035354614, 0.831002950668335, 2.5727999210357666, 0.6849489212036133, 1.2531280517578125, 0.15194383263587952, 0.2276483029127121, 2.2155189514160156, 0.42834165692329407, 1.5121345520019531, 1.635521411895752, -0.22467023134231567, 1.7745959758758545, 1.0318976640701294, -0.6126760840415955, -0.889924943447113, -0.8377166390419006, -0.3628994822502136, 0.3059421479701996, -0.3604338765144348, 1.3472061157226562, -1.1244934797286987, -0.7770294547080994, -1.265818476676941, 0.14016862213611603, -0.2536904215812683, -0.1740119606256485, -0.48233723640441895, 0.047484494745731354, -0.9867989420890808, -0.5692941546440125, -0.9638115167617798, -0.4643842875957489, -0.1450149416923523, -1.0153154134750366, -0.9069439768791199, 0.5303090810775757, 0.1534215360879898, -1.1200025081634521, 0.7972589135169983, 1.060244083404541, -2.2090256214141846, -1.1664934158325195, -0.01031811535358429, 0.15791471302509308, -0.7957915663719177, 0.261198490858078, 0.31782305240631104, 0.1592831015586853, 0.9355635046958923, 0.20938417315483093, -1.1507922410964966, -0.13753458857536316, -0.4287773072719574, 1.2102630138397217, -0.36517858505249023, -0.7931997776031494, 0.6696242094039917, 0.05214325338602066, -0.3560854494571686, 0.49409976601600647, 1.5296216011047363, 0.8123978972434998, -1.1047582626342773, -1.863715410232544, -1.0429452657699585, 0.36002668738365173, 0.14674042165279388, 1.0519704818725586, -0.6963188052177429, 1.3247017860412598, -0.9481844902038574, 0.9551901817321777, -0.08410382270812988, -0.4540943205356598, -0.6090648174285889, 0.24116294085979462, -0.124352365732193, -1.3251999616622925, -2.5391194820404053, -1.8686784505844116, 1.378986120223999, -1.4281988143920898, -1.230076789855957, -0.1733512431383133, 0.7800039649009705, 0.8696750998497009, -0.10035905987024307, -0.015693634748458862, 0.2163647711277008, 0.5847485065460205, 0.613976776599884, -0.13781072199344635, -0.33645159006118774, 0.7970442771911621, 0.9117083549499512, 0.2318735271692276, -0.5057364106178284, 0.11155035346746445, -1.673226237297058, -0.23696254193782806, 0.1663663685321808, 1.2295420169830322, -0.0679769292473793, 0.30184200406074524, -0.7983502745628357, -0.03277365490794182, -1.4885348081588745, 0.35273003578186035, 0.2785080373287201, 1.3538228273391724, -0.17213216423988342, -1.5206935405731201, 1.4861421585083008, -1.362923502922058, -0.43077144026756287, -1.3438024520874023, 0.005988063290715218, 1.5500750541687012, -0.5989853143692017, -0.5877674221992493, -0.5324847102165222, 0.6867451667785645, 0.3947920501232147, -0.866199791431427, 0.23968587815761566, -0.1961795836687088, -0.3890911042690277, -0.8630679845809937, 0.22438131272792816, -1.196010708808899, -0.02444462478160858, -0.8605937957763672, -1.0974788665771484, 0.8706283569335938, 0.7564175724983215, -0.2925857901573181, 0.40368691086769104, 1.3742609024047852, -1.203454613685608, -0.36391985416412354, 0.16470913589000702, -0.15374243259429932, 0.37325918674468994, -0.6834466457366943, -0.359881728887558, 0.042643822729587555, 1.4124621152877808, -1.031395435333252, 0.9929124116897583, -2.1172709465026855, 0.20711281895637512, 0.36269712448120117, -2.4301981925964355, -1.2284859418869019, -0.5054289102554321, 1.0604655742645264, 1.6322879791259766, 1.0397833585739136, 0.09638825803995132, 1.0552998781204224, -0.7841755747795105, 0.2832086682319641, 1.12962806224823, -0.7287349700927734, -0.6204398274421692, -0.3135809898376465, 0.04693193361163139, -0.49595722556114197, -0.06225069239735603, -1.650043249130249, 0.21746128797531128, -0.9643715023994446, 0.768333911895752, -1.035224199295044, -0.876966655254364, 0.8520678281784058, -2.0230886936187744, -0.17549289762973785, -1.1679071187973022, 0.24623261392116547, 0.08407004922628403, 0.9455165266990662, -1.56101655960083, -0.9170994162559509, -0.4813215732574463, 1.7272950410842896, 0.1231217011809349, -0.7756474614143372, -0.310468465089798, -0.8766818642616272, 1.957241177558899, -0.3042502999305725, 0.7185678482055664, -0.7589918971061707, 0.6739634871482849, -0.03048143908381462, -0.29444536566734314, -0.12845531105995178, -0.6632916927337646, -0.7586209177970886, -0.4718674421310425, 0.5564046502113342, 0.25722411274909973, 0.6088253259658813, -1.147491693496704, 0.7754155993461609, -1.4726868867874146, 0.661571204662323, -1.7127923965454102, 0.4394765794277191, -2.399585485458374, 0.39030539989471436, -0.9054353833198547, 0.21616560220718384, -0.29269489645957947, 1.3048651218414307, 1.2441424131393433, -0.18189862370491028, -0.7012102603912354, 1.2435529232025146, 0.9328538179397583, 0.7886918783187866, 1.555896520614624, -0.546488344669342, -0.7637581825256348, 0.1317531019449234, 1.2418811321258545, 0.08903754502534866, -0.9636462330818176, 1.2709380388259888, 0.41464030742645264, -1.7851908206939697, -0.7275315523147583, -0.23643384873867035, -0.8206360340118408, 0.6528871059417725, 1.377912998199463, 0.005222464445978403, 0.6091731190681458, -0.4175070524215698, 0.16123993694782257, 1.0738779306411743, -0.17468301951885223, 1.0546128749847412, 1.185683250427246, 0.12052033841609955, -2.5920016765594482, 1.5601475238800049, -0.578337550163269, 0.48246538639068604, 1.16460382938385, -2.3535516262054443, -0.44600144028663635, 1.034062147140503, -1.9607194662094116, 1.822479009628296, 0.5590938329696655, 0.8739319443702698, 1.3405393362045288, 0.7530739903450012, -0.3519417941570282, -0.8118260502815247, 0.18567030131816864, 1.096529483795166, 1.0890991687774658, 0.3315757215023041, -2.1211421489715576, 0.007597034331411123, -0.0774117037653923, 0.2154475599527359, -0.7523491382598877, 0.4597318768501282, -2.4321465492248535, 1.3184490203857422, 0.4711059629917145, 2.11238956451416, -1.4729437828063965, -0.031746622174978256, -0.6746812462806702, 0.6470149755477905, 0.06546705216169357, 1.74289071559906, 0.8625389933586121, 0.02665255218744278, 0.07616113126277924, -0.4292895197868347, 0.763380765914917, 1.3606929779052734, -1.416663646697998, -1.2471113204956055, 0.7435508966445923, 0.7413820624351501, -0.7105478048324585, -0.5308656692504883, 1.1556943655014038, -0.4757538139820099, 1.1169030666351318, -2.19006085395813, -0.13446317613124847, -0.4060099720954895, -0.8608322739601135, 0.1753559708595276, 0.7793733477592468, -0.10611318796873093, -1.076192855834961, -0.3229256868362427, 1.8652806282043457, -0.6405158042907715, 0.699954092502594, 0.06337374448776245, -0.7433082461357117, -0.798280656337738, 0.2565430998802185, -0.23211215436458588, -1.4143632650375366, -2.2270519733428955, 1.2159004211425781, 0.04150109365582466, -0.6139360070228577, -0.4437246024608612, 2.345428228378296, -0.8969794511795044, -0.1370212286710739, 0.1088285744190216, 1.1743587255477905, 1.5438714027404785, 1.8320964574813843, -0.2194797247648239, 0.7152698636054993, 1.5642892122268677, -1.3519694805145264, 0.9472238421440125, -0.5017879605293274, 0.9361850023269653, 0.3733363449573517, -0.4547913074493408, -0.15660081803798676, -0.1670466959476471, -0.3525386154651642, 0.10559936612844467, -0.6307696104049683, 1.6439858675003052, -0.0021007645409554243, 0.8437350392341614, 0.24358266592025757, -1.4794366359710693, 1.2293460369110107, 0.49274909496307373, 0.19869305193424225, -1.247583270072937, 0.48187145590782166, 0.9439544081687927, 0.682049572467804, -0.6119599938392639, 0.6771251559257507, 0.14637403190135956, 1.151315450668335, 1.600812315940857, 0.5347517132759094, 0.06448080390691757, -1.2994775772094727, 0.09726754575967789, -0.6420168280601501, 0.43081173300743103, -1.0212383270263672, -0.8281762003898621, -0.6760307550430298, 0.49325111508369446, 0.5692182183265686, 0.8014026880264282, -0.41615593433380127, -0.4967959225177765, 0.8000137209892273, -0.07281052321195602, -0.8804929256439209, -0.20559367537498474, 1.094098448753357, 0.4728540778160095, 0.6738728284835815, -1.7828233242034912, 0.6268141865730286, 0.624893069267273, 0.14878888428211212, 0.7941465973854065, 1.5810019969940186, -0.32284775376319885, 1.9789780378341675, 0.114835225045681, 0.15075074136257172, -2.0519561767578125, -0.39778372645378113, 0.007295995019376278, -0.8001174330711365, 1.333725929260254, -0.8424106240272522, 1.738032341003418, -2.504863739013672, -0.3506876528263092, -2.892200469970703, 1.1250635385513306, -0.5626453757286072, -0.17737923562526703, 0.6224374771118164, 0.9447112083435059, 2.0579962730407715, -1.1445670127868652, -2.325528860092163, -0.4770873785018921, -0.13227464258670807, 2.4025208950042725, 1.364486813545227, -0.30091041326522827, -1.386153221130371, -0.23894254863262177, 0.43948104977607727, -1.2794222831726074, -0.03745422884821892, -0.4490124583244324, -0.18283595144748688, 0.576560378074646, -0.00930300634354353, -0.5918751955032349, -0.5314499735832214, -0.5371014475822449, 1.4680585861206055, -0.190202534198761, -1.4858897924423218, -0.10782577842473984, 1.3395458459854126, 1.2255988121032715, -0.3122458755970001, 0.027922889217734337, -0.4161938726902008, -0.0751887857913971, -0.6934830546379089, -0.10004586726427078, -0.39439547061920166, 0.4473341703414917, 1.5268644094467163, -1.2880147695541382, -0.2879677712917328, -0.3772728443145752, 1.749213695526123, -1.9262592792510986, -0.3779105544090271, -0.6994211673736572, -1.2481091022491455, -1.0917192697525024, 0.5384536385536194, -1.1392351388931274, -1.9496158361434937, -1.146577000617981, -0.617824912071228, -0.997759222984314, -1.301230788230896, -0.029198095202445984, -0.10233689844608307, -0.8157586455345154, 0.3042321801185608, -0.2997196912765503, 0.12547242641448975, 0.6227982044219971, 0.5471245050430298, 0.6491696238517761, -0.728459358215332, 1.7922617197036743, 1.1093063354492188, -0.506007730960846, -0.2314036637544632, -0.6710328459739685, -0.38438308238983154, 1.3234994411468506, 0.8458675146102905, -0.6315253376960754, -1.1166208982467651, -0.3492979407310486, -0.9092738032341003, -0.28355103731155396, 0.6612783074378967, 0.9012936949729919, -1.1062419414520264, 0.9098753333091736, 0.09542616456747055, -0.5966455936431885, 0.3784802258014679, -1.478385090827942, -1.3685903549194336, 1.2867825031280518, 0.41112038493156433, -1.4910509586334229, -1.8999180793762207, -1.4163211584091187, -2.100163221359253, 0.8256199359893799, 1.0291526317596436, -0.3096340000629425, -1.1387690305709839, -1.04237699508667, -0.840808093547821, -0.8734933137893677, -0.3892545700073242, 2.0125956535339355, -1.7106690406799316, -1.9105501174926758, -0.9979693293571472, -1.088224172592163, 0.5306689739227295, -1.2412201166152954, -0.5281221270561218, -1.0046766996383667, -0.15601420402526855, -1.069831132888794, -1.0704929828643799, -0.005020458716899157, -1.957387089729309, 1.9708623886108398, -0.6339045166969299, 0.8403235077857971, -0.34812691807746887, 0.5232506990432739, 0.3601003885269165, 0.62397301197052, -0.549868643283844, -1.2750000953674316, -0.5476012825965881, -1.1224230527877808, -0.661266028881073, 0.173564150929451, 0.08130568265914917, -0.1221708282828331, 0.8595555424690247, 0.9835416674613953, 1.1160129308700562, 0.3459234833717346, 0.5600408315658569, -0.26553091406822205, 1.1304045915603638, 0.3538041412830353, -0.647575318813324, -0.3625459372997284, -0.7939348816871643, -0.1261928677558899, -0.35235029458999634, 0.21159727871418, 1.4003145694732666, 0.8303209543228149, -1.157480239868164, 0.5261712074279785, 0.989730715751648, 0.7856330871582031, 3.4512696266174316, -1.5390071868896484, -2.9025988578796387, 1.154664158821106, 0.5796216130256653, 0.9028530120849609, 1.438815712928772, 0.3100317120552063, -1.1003878116607666, 0.1988115906715393, 0.10160315781831741, -0.6370859742164612, 0.6375229954719543, 0.3301777243614197, -0.425851970911026, 0.9573779702186584, -2.524747610092163, 0.2627514600753784, 0.42759940028190613, 0.25220105051994324, -0.647849977016449, 1.2938594818115234, -0.4572506546974182, -0.6257063746452332, 1.151114821434021, -0.004940370097756386, 0.6222512722015381, -1.129290223121643, -0.6347482800483704, 1.028415322303772, -0.3093872666358948, 0.25438427925109863, -0.4028880298137665, -0.07706823199987411, 0.20402510464191437, 1.2453644275665283, -0.4346795678138733, 0.6256517767906189, -0.6472955942153931, -1.0501006841659546, -0.046907663345336914, -1.6421663761138916, -0.5725464224815369, 0.21088807284832, -0.14610058069229126, 1.5437442064285278, 0.6266505122184753, 0.16379454731941223, 0.32246872782707214, 0.4823662042617798, 1.0031954050064087, 0.42924168705940247, 0.7410284280776978, -0.17324790358543396, -1.8528016805648804, 0.2705916464328766, 0.6869301199913025, 3.1596155166625977, 0.4111200273036957, 0.9816949367523193, -0.22979214787483215, -0.9334706664085388, 1.3029839992523193, 0.4667564332485199, 2.870602607727051, -0.5239275097846985], [0.9567946195602417, 0.6952137351036072, 0.8618718981742859, 2.8976938724517822, 0.6053005456924438, -0.6619364023208618, 0.2933513820171356, -0.24888238310813904, 1.2932935953140259, -0.3642963767051697, -1.8917988538742065, 0.8313365578651428, 0.7949804067611694, -0.5480496883392334, -1.106656551361084, 0.44716453552246094, -1.0151355266571045, 0.956612765789032, 0.4887789487838745, 2.3645896911621094, 0.756140410900116, 1.2666102647781372, -0.5376666188240051, -0.7057791948318481, 0.8169969320297241, 0.9391490817070007, 0.46721121668815613, -1.7487584352493286, 0.19435346126556396, 0.12382771819829941, -0.028548510745167732, -0.8117250204086304, -1.4913606643676758, -0.4166337549686432, -0.7814242243766785, 1.4962466955184937, -0.03743038699030876, -0.48719218373298645, -0.061201855540275574, 1.353734016418457, -0.6032496690750122, -0.3090575337409973, -0.2439577430486679, 0.7418935894966125, 0.3346591591835022, -1.2708266973495483, 0.059472620487213135, -0.4101404845714569, 0.8312889337539673, 0.5914397239685059, -0.1316063106060028, 2.3118741512298584, -0.08617990463972092, -1.2994199991226196, -0.5779507756233215, 1.4418641328811646, 1.3311771154403687, 0.2266968935728073, 0.03301994875073433, 1.4604874849319458, 0.22263287007808685, -0.13141264021396637, -1.6876275539398193, 0.6177065372467041, -0.5547541379928589, 0.20978069305419922, -0.6793473958969116, -1.473711609840393, 0.6463109254837036, -0.9623466730117798, -1.2342524528503418, 0.7523744702339172, 1.2046531438827515, 0.5469241738319397, 1.4952785968780518, 0.3419733941555023, 0.4995145797729492, -0.8410871028900146, -0.7803871035575867, -1.2974627017974854, 0.6456425189971924, -0.30882811546325684, -1.0668632984161377, -1.8989636898040771, -0.9893909096717834, 0.46933606266975403, 0.5413779616355896, 1.7485764026641846, -0.435256689786911, -0.03120327740907669, 0.052594829350709915, 2.0980663299560547, 0.3387911319732666, -0.22795742750167847, 0.1813148409128189, 0.02790898270905018, -0.47213685512542725, 0.5942472219467163, -2.451403856277466, -0.5238995552062988, -1.7078696489334106, -1.122184157371521, 1.389809012413025, -0.15521784126758575, -0.9231666326522827, 0.7995324730873108, 0.4147591292858124, 1.3491861820220947, -0.625164806842804, -1.0022118091583252, -0.06869561970233917, 1.4683102369308472, 0.49692898988723755, -1.0183625221252441, -0.7676828503608704, -0.17004811763763428, -1.3580589294433594, -0.5845908522605896, 1.7898136377334595, -2.3276944160461426, -2.6899025440216064, -0.2136581540107727, 0.16970095038414001, -0.28341755270957947, -0.29923221468925476, 1.0752850770950317, 0.5257431864738464, 0.7507777214050293, 1.9947069883346558, -0.11974089592695236, -0.34710246324539185, 0.5394787788391113, 0.778183102607727, -0.21836748719215393, 3.284541606903076, 0.5056890845298767, -0.011769667267799377, -0.7530410885810852, 1.0558617115020752, -1.498547077178955, 0.9296219348907471, -0.940261721611023, -0.15183471143245697, 0.2857348322868347, 0.9232084155082703, 0.4578646123409271, 1.255138874053955, 2.02847957611084, -0.7157734036445618, -0.5043495297431946, -0.8414396643638611, 0.09041383862495422, 1.1831828355789185, 0.01812555640935898, 0.49689847230911255, 0.42257460951805115, 0.34010201692581177, 0.9468977451324463, -1.5764342546463013, -0.3336816430091858, 1.2896652221679688, -1.3806886672973633, 0.7738197445869446, 0.975945234298706, -0.8628867268562317, -0.7205439209938049, -0.812084436416626, -1.4305516481399536, -1.4458682537078857, 0.1398991495370865, 1.164153814315796, 0.2510576546192169, 0.5661298632621765, 1.9221850633621216, -0.39462804794311523, -0.24326658248901367, -0.7144606709480286, 0.4304860830307007, -0.3852095305919647, 0.26684486865997314, 0.3648759722709656, 0.6651426553726196, -0.15744563937187195, 1.848196029663086, -0.1618482917547226, 1.366085410118103, -0.7982425689697266, -1.0849888324737549, -0.93928462266922, 0.21048498153686523, 0.2069232016801834, 1.0856907367706299, 0.8589657545089722, -0.30906960368156433, 0.9220098853111267, -0.3912504017353058, 1.0172336101531982, 0.7219575643539429, 0.35932767391204834, 0.39646416902542114, 0.6843599677085876, 1.6389400959014893, -0.45176222920417786, -1.0028975009918213, -2.1583616733551025, 1.5333538055419922, -0.6040118336677551, -1.1020132303237915, -0.5836490392684937, 0.6401255130767822, 1.3835411071777344, -0.41753727197647095, 2.463825225830078, -0.47607070207595825, 0.25575676560401917, 0.23614370822906494, 1.066038966178894, -0.4093243181705475, 0.7842997908592224, -1.1543406248092651, 0.16955213248729706, 1.04758620262146, -1.7547391653060913, -0.2771197259426117, -0.5123735070228577, -2.322385549545288, -0.9771290421485901, 0.29399269819259644, -0.2837822735309601, -0.9135276675224304, -1.3239333629608154, -0.10348980128765106, -1.3662484884262085, -1.4813482761383057, -0.3775350749492645, -1.1659088134765625, -1.0653010606765747, 0.7161062359809875, 1.1156054735183716, 0.038834016770124435, -1.2404413223266602, 0.48526427149772644, -0.012678710743784904, -1.3628075122833252, -1.1993719339370728, -1.446337103843689, -0.4028627872467041, 1.234782099723816, -0.06871631741523743, 0.12140029668807983, 1.0321353673934937, -0.30840781331062317, 0.4414229094982147, -0.37808147072792053, 2.139599084854126, 0.11232440918684006, 0.4688485264778137, -1.6892393827438354, -0.3062463700771332, -1.3922195434570312, -1.6377826929092407, 0.40405330061912537, 0.42711883783340454, 1.1908073425292969, -1.6057871580123901, -2.1097939014434814, -0.22408916056156158, 0.21638800203800201, 0.9641279578208923, 0.17174287140369415, -1.6780580282211304, -0.38953709602355957, 1.6470324993133545, 0.05787453427910805, 0.1933884173631668, -1.4867490530014038, 0.9320915937423706, 0.4651704728603363, 0.18389669060707092, -0.7991414070129395, -0.17253588140010834, 0.8678668737411499, -0.8273237943649292, 1.1150213479995728, -0.9557788968086243, 1.0246973037719727, 1.9508459568023682, -1.4323097467422485, -1.0405018329620361, 1.3512619733810425, -0.48361071944236755, 0.3919556438922882, 1.4453808069229126, -0.423142671585083, -0.465408593416214, -0.871034562587738, 0.3018476963043213, 0.8970308899879456, -1.6470612287521362, -0.25651273131370544, 2.038362503051758, 0.47311705350875854, 0.4473213851451874, -1.2517824172973633, 0.3374474048614502, 0.32102060317993164, -0.7344105839729309, -0.9057405591011047, -0.9742060899734497, 1.3637827634811401, -1.6926233768463135, 0.9584774971008301, 0.2310798466205597, 0.6030563116073608, 1.4791491031646729, -0.330166757106781, 0.5655209422111511, 0.6748449802398682, -0.2648431360721588, 0.4010135531425476, 0.09938511997461319, 1.0625146627426147, 0.5753188729286194, 0.5030372142791748, -1.3844243288040161, 0.6271563768386841, -0.7678835988044739, 1.246826410293579, -0.636903703212738, -0.27143678069114685, 0.6643668413162231, -1.1966646909713745, 0.24891622364521027, -0.7638837099075317, 0.48375752568244934, -0.31740590929985046, 0.9487063884735107, 0.2948363125324249, -0.7021647691726685, -0.20266981422901154, -0.029799042269587517, 2.551997184753418, -0.0451466366648674, -0.4727526605129242, -1.1489895582199097, -0.6683240532875061, 0.8209946751594543, -0.30309340357780457, -0.3210996389389038, -2.300985336303711, -0.42259418964385986, 1.1992783546447754, 2.5385549068450928, 0.24831318855285645, 0.15491075813770294, 0.31208452582359314, -0.5293709635734558, -0.9816842675209045, -0.2517193853855133, -1.897497534751892, -0.9826895594596863, 0.13948838412761688, 0.49482056498527527, -0.3792125880718231, 0.1720103919506073, 0.7274613380432129, -0.46808138489723206, -0.7698794603347778, -0.07979822158813477, -0.49636414647102356, -0.15067054331302643, -0.5017465949058533, 0.6789096593856812, -0.7390612363815308, -1.7712101936340332, -0.5315018892288208, 0.01600838638842106, -0.36808088421821594, 0.29474470019340515, 0.27100101113319397, 0.2957450747489929, 0.9733112454414368, -0.2034914791584015, -2.0997843742370605, 1.4886367321014404, 1.5252647399902344, -0.12452487647533417, -0.0875273123383522, 0.129765585064888, -0.19360460340976715, -0.6062972545623779, 1.3776625394821167, -0.24977916479110718, 1.26194429397583, -0.19450634717941284, 1.7372349500656128, 1.1093777418136597, -1.8213287591934204, 0.5263077020645142, 0.222168430685997, 0.7057735323905945, -0.04175299406051636, -0.6385812163352966, 0.30987095832824707, -0.3940601050853729, -0.34865501523017883, -1.5260964632034302, 0.09725655615329742, -1.6024701595306396, -0.6410605311393738, 0.3730778992176056, 1.235548973083496, -1.7563496828079224, 2.1049017906188965, 1.8207807540893555, 1.1684147119522095, -0.14132201671600342, -0.8535848259925842, -0.18977604806423187, -1.5750293731689453, -1.1450928449630737, 0.41148853302001953, -1.3189314603805542, 0.4252426028251648, 1.7161445617675781, -1.2615971565246582, -0.1412719041109085, -1.367496371269226, -0.022931430488824844, -1.3690005540847778, 0.48785683512687683, 1.304948329925537, -0.6345768570899963, -0.0732991024851799, -0.6020494103431702, 0.025290129706263542, 1.1004788875579834, -0.9259526133537292, 1.700606107711792, -0.04481610283255577, 1.222843050956726, -0.35327163338661194, -0.5707734227180481, -1.8267878293991089, 0.7173910737037659, 0.5697627067565918, 1.2798993587493896, -1.031026840209961, 0.6116626262664795, -0.43327295780181885, -0.8395070433616638, 0.13588547706604004, 1.2015994787216187, 1.1432408094406128, -0.6710463762283325, -0.6458990573883057, -1.4739341735839844, -0.6636822819709778, 1.6679118871688843, -0.7469696998596191, -0.008070658892393112, 1.862951397895813, -0.9000688195228577, 2.5177998542785645, -0.5797937512397766, 0.03873198479413986, 0.6010869741439819, 0.5729983448982239, -0.34916582703590393, -0.200588196516037, 0.8140038847923279, 0.013690377585589886, 0.4429759383201599, -0.28637975454330444, 1.9677382707595825, 0.673822820186615, 1.9995630979537964, -2.0516114234924316, 0.6157239079475403, 1.372550129890442, -0.3904237449169159, -1.1042534112930298, -0.5766967535018921, -0.3028143346309662, 0.5256681442260742, -1.4770194292068481, 0.7734483480453491, -1.8698949813842773, -0.28395506739616394, -0.8320801854133606, 0.6098117828369141, -0.7856296896934509, -0.5310138463973999, -1.1087300777435303, 0.025068318471312523, 0.2802991569042206, 1.9757453203201294, 0.33630406856536865, -2.072643518447876, -1.5597862005233765, 2.117835283279419, 0.4004170894622803, -1.217477798461914, -0.3079366385936737, -0.10819555073976517, 0.32838213443756104, -0.3812883794307709, -0.15001384913921356, -0.20310106873512268, -0.21516025066375732, -1.3331090211868286, 0.24723494052886963, 0.5847398042678833, -0.538518488407135, -0.6380720734596252, 0.44002416729927063, -0.1415533423423767, -1.2874559164047241, 1.7880737781524658, 0.7948921918869019, -1.9859232902526855, -0.8143763542175293, -0.35213470458984375, 0.26824963092803955, -0.14294633269309998, 2.1279356479644775, 1.0630100965499878, 0.973617672920227, 1.074855923652649, 0.3230699598789215, -0.8186022043228149, 1.2518234252929688, -0.288379430770874, 0.12266407161951065, -0.829458475112915, -0.938443660736084, -0.7884724736213684, -0.5889685750007629, 0.5138120055198669, 0.08988041430711746, -1.6510874032974243, -0.29573702812194824, 0.3555222153663635, 1.5675102472305298, -0.07466895133256912, -1.1846994161605835, 0.5592623949050903, 2.3111448287963867, 1.2785793542861938, 1.3617901802062988, 0.3449753522872925, 0.6336650848388672, -0.4265806972980499, -0.155196875333786, -0.19911229610443115, -0.6112945079803467, -0.8610360026359558, 0.9410646557807922, 0.8158081769943237, -0.5341739058494568, 1.1061040163040161, -0.8206332921981812, -1.6379002332687378, 0.8097161650657654, 0.3492204546928406, -0.36200669407844543, -1.0334758758544922, 0.011373650282621384, -0.40439411997795105, 0.9315853714942932, 0.7135065197944641, -0.05589606240391731, -1.0437846183776855, -0.463352769613266, 0.8471693396568298, -0.31100282073020935, -0.2254052460193634, -0.44236648082733154, -0.7034327387809753, -0.1428196281194687, -1.4493228197097778, 0.1440018266439438, 0.6947767734527588, 0.18639308214187622, -0.20485347509384155, -0.453792542219162, -1.5338492393493652, 0.24340736865997314, -1.2276883125305176, 1.7134649753570557, 0.799761950969696, 0.29273107647895813, 0.9813448786735535, 0.2777003347873688, 0.6281319856643677, -0.9476408958435059, -0.5424408316612244, 0.47462135553359985, -0.02934063971042633, -0.5549945831298828, -0.9790918231010437, -0.33933231234550476, -0.8041516542434692, -0.409941166639328, -0.6199567317962646, 0.22454611957073212, 1.2398627996444702, 1.4066033363342285, 0.1945437788963318, -1.2725763320922852, 0.19212405383586884, -0.27877524495124817, 1.0851593017578125, 0.056107115000486374, 0.3183033764362335, 0.6497384905815125, -0.4915502667427063, -0.8284127116203308, 1.2846730947494507, 0.35397645831108093, 0.897545337677002, 0.39358651638031006, -0.3382667303085327, 0.5084872841835022, 0.14576323330402374, 0.14543525874614716, -1.1058582067489624, -1.6270242929458618, -0.6370661854743958, -0.2982282340526581, 1.1665871143341064, -0.25349777936935425, -0.36252230405807495, 0.3912888467311859, -0.47248250246047974, 0.6312558650970459, -0.7735427021980286, -0.24720558524131775, 0.1853857934474945, 0.5515624284744263, -0.9457266926765442, -0.963250994682312, 0.4268517792224884, -0.2644355893135071, 0.1228259801864624, 0.7057867646217346, 1.6115361452102661, 1.419281244277954, 1.1029337644577026, -0.6850783228874207, -0.7162573933601379, 0.6177370548248291, -0.8076541423797607, -0.42986902594566345, 0.6074408888816833, -0.8448891043663025, -0.42121222615242004, 0.8753450512886047, -1.4227359294891357, -0.2962961494922638, 1.6478986740112305, -0.7186927199363708, -0.10241815447807312, 0.731512188911438, 0.34732839465141296, 0.8470874428749084, 1.30912184715271, 0.009282346814870834, -0.6521388292312622, -0.028223853558301926, 0.36917373538017273, 1.490583062171936, 1.4782313108444214, 0.5387483239173889, 0.5319094657897949, -0.3519175350666046, -1.157142996788025, -0.1818099319934845, 1.230417013168335, 0.22893230617046356, 1.3176372051239014, 0.5557355880737305, -0.447603702545166, -1.3587895631790161, 0.5764702558517456, -0.6917591094970703, 0.544131338596344, 2.1654319763183594, 0.3172037601470947, -0.10843689739704132, 1.409162998199463, -0.6876139044761658, 1.3308659791946411, 0.7423617243766785, 1.5303436517715454, 0.056050658226013184, -0.12703938782215118, -0.6686424612998962, -0.9615110158920288, -0.6453871130943298, -0.2818203866481781, 0.33602645993232727, 1.1112562417984009, 0.39882713556289673, -0.9390466213226318, 0.9570230841636658, 2.2602906227111816, -0.15398474037647247, 1.4682973623275757, 1.3319075107574463, 0.0254230760037899, -0.03627482056617737, -1.3652944564819336, 0.7437925338745117, 0.7863937020301819, 0.3020944595336914, -0.738220751285553, 0.10145597159862518, 1.1401876211166382, 0.5526822209358215, 0.5174317955970764, -0.06377656012773514, -0.3564703166484833, 1.082079291343689, 0.6625916957855225, 0.4819575548171997, 0.5327237844467163, -0.10498563200235367, -0.5980933308601379, 0.17184194922447205, 1.7087520360946655, 0.21803881227970123, 1.5338443517684937, -1.7113629579544067, -1.4826362133026123, -1.1793444156646729, -0.5798024535179138, 0.08896956592798233, 2.3518457412719727, -0.3363426923751831, -1.151593565940857, -0.9631964564323425, 0.7658621072769165, -0.18145491182804108, 0.15582430362701416, -0.4586418867111206, -0.6584973931312561, -1.6878420114517212, 0.023096172139048576, 0.9739840030670166, -0.26809296011924744, -0.513607382774353, 0.026508091017603874, 0.11282290518283844, 1.9859287738800049, 0.6198532581329346, -0.7671512365341187, 0.8628149628639221, -0.7119492292404175, 1.1310985088348389, -0.1571519821882248, 0.15731285512447357, -1.5144321918487549, 0.7830497026443481, 0.287152498960495, 0.642703115940094, -1.258133053779602, 0.3922080099582672, 1.077062726020813, -0.09739905595779419, -0.16453616321086884, -0.6933184266090393, 0.78812175989151, 1.9151314496994019, 0.5272108316421509, -0.12211012095212936, 1.532599687576294, 0.8166543245315552, -1.2845982313156128, 0.2237478494644165, -2.698678493499756, -0.09846267849206924, -0.7985165119171143, -0.4148388206958771, -0.49872827529907227, -0.40816056728363037, 1.8310343027114868, 0.5455939769744873, -1.3967673778533936, -1.6459637880325317, 0.027023877948522568, -0.6489524841308594, 0.29293179512023926, 0.6309832334518433, 0.15106211602687836, 0.8338268399238586, -0.23035743832588196, 2.36808443069458, 0.47904276847839355, -1.1839529275894165, -0.7959925532341003, -0.5262244939804077, 0.8249072432518005, 1.2672796249389648, -0.42616912722587585, 0.936244010925293, 0.022313447669148445, -0.6606166958808899, -1.2371209859848022, -0.13779032230377197, 0.25204116106033325, -1.1593388319015503, -0.9574230313301086, 0.08549661934375763, 0.9147444367408752, -0.40736088156700134, 1.1627023220062256, -0.34626686573028564, -0.0937834307551384, -0.069363534450531, -0.9516777396202087, 0.1866772472858429, 0.537735641002655, -1.1805856227874756, 2.1532838344573975, -1.7084579467773438, -0.7808976769447327, 1.5284233093261719, -1.4539575576782227, 0.46153196692466736, -1.2784388065338135, -0.005950374063104391, -0.46261072158813477, 0.7808897495269775, -1.5905879735946655, -0.763616681098938, -1.0683354139328003, 0.8049783706665039, 0.687175989151001, 0.748447060585022, 0.15679588913917542, -0.21116793155670166, 1.2435064315795898, 0.8412594795227051, 0.8884024620056152, -1.3398910760879517, 1.6423345804214478, 0.8211197853088379, 1.4553911685943604, -0.2859247028827667, -1.4317681789398193, -0.6617849469184875, 1.4151214361190796, 0.1090310662984848, 0.7382544875144958, -0.28795453906059265, 0.7406296730041504, 1.332261323928833, -0.06973553448915482, -0.1706533133983612, -0.1034596860408783, -0.43682610988616943, 0.47245892882347107, -1.5554232597351074, 1.5986946821212769, -1.7138683795928955, -0.9467806220054626, 0.355966180562973, 1.141677737236023, -0.21048909425735474, 0.6578408479690552, -1.2699322700500488, 0.8440515398979187, 0.7907324433326721, 1.280663013458252, -0.37809035181999207, 0.09485973417758942, -0.6701679229736328, -0.7876319885253906, 0.7842977643013, 0.06210223212838173, 0.872422456741333, 1.062536358833313, 0.09964606165885925, 0.3857155442237854, 1.654700517654419, 0.037213630974292755, -0.11888618767261505, 0.8980093002319336, 2.1365299224853516, 1.959277868270874, -1.3308757543563843, 0.18620379269123077, 0.8498969078063965, -0.8397206664085388, -0.12034963071346283, -1.039940357208252, 0.4938122630119324, 0.06036333367228508, -0.3577064573764801, -1.4711384773254395, 2.1786458492279053, 0.18565596640110016, -0.9170103073120117, 0.31271272897720337, 2.0482289791107178, 1.358078122138977, -0.9078978896141052, 1.487166404724121, 0.3926505744457245, -0.9850041270256042, -1.4227300882339478, 1.3560409545898438, 1.5478923320770264, -0.6433413624763489, 0.8050300478935242, 0.0004884309601038694, -0.21783913671970367, 0.5156617164611816, 0.5908398628234863, 1.1261224746704102, -0.6614906191825867, 1.6373776197433472, 0.5955976843833923, 0.35242941975593567, 0.765021800994873, -0.6593303680419922, 0.7614782452583313, -2.800469398498535, 1.6822205781936646, -0.7544561624526978, 0.014241408556699753, 2.378406047821045, 0.2295566350221634, -0.5597190856933594, 1.4424115419387817, -0.1103394627571106, -0.05240647867321968, 0.05030963197350502, -1.5792045593261719, -0.755021333694458, -1.3908766508102417, 0.461932510137558, -0.27495983242988586, 1.6796784400939941, 0.2929927408695221, 1.6575418710708618, -1.363379955291748, 0.2312185913324356, -1.9280585050582886, -0.1451953947544098, 1.1126946210861206, 2.4729344844818115, 2.1017346382141113, -0.20473238825798035, -0.1046653762459755, 0.4397425949573517, -0.2279009371995926, -1.1617770195007324, 0.6765115261077881, -0.2802426517009735, -0.35258933901786804, -0.38832250237464905, 0.7704915404319763, -1.3487132787704468, -0.4031521677970886, 1.835986852645874, 1.3395754098892212, -0.4950428605079651, 0.7079386115074158, 0.4467376470565796, -0.25630316138267517, 2.2242302894592285, -2.058234930038452, 0.1269315481185913, 0.8634473085403442, 0.8695652484893799, -0.818760871887207, 0.45645010471343994, -0.6790090203285217, 0.07666786760091782, 1.330954670906067, -0.633854329586029, 0.20424498617649078, 0.5151482224464417, 0.9337590336799622, -0.5849557518959045, -0.4494003653526306, -0.5388598442077637, 0.5928568840026855, -0.557639479637146, -1.2041306495666504, 0.0863286703824997, -0.37353700399398804, -1.0919826030731201, 1.1369991302490234, 0.6694377660751343, -0.41828271746635437, 1.5708839893341064, 0.6492675542831421], [-0.40483757853507996, -1.2620583772659302, 0.9215013980865479, 0.21651867032051086, 0.626528799533844, -1.8241509199142456, 0.24492517113685608, 0.9274563193321228, 0.49582087993621826, 0.14554819464683533, 1.7212311029434204, -1.1164376735687256, -0.729290246963501, -0.8095458149909973, 0.0011369066778570414, 1.6618282794952393, -0.8344184160232544, 1.3840559720993042, -1.3349478244781494, -1.2321175336837769, 1.7349046468734741, 0.5400844216346741, -0.7645418047904968, -0.0022941171191632748, 0.7688221335411072, 1.6288284063339233, 2.1793055534362793, -0.9407551288604736, 0.5839870572090149, -1.160752773284912, -1.8511416912078857, 1.6415830850601196, 0.8325307369232178, 0.35853755474090576, 0.4335934817790985, 0.6737668514251709, -1.0392701625823975, -0.8624643683433533, -1.0518956184387207, -0.2684376537799835, 1.3545762300491333, -1.79122793674469, -0.09912890940904617, 0.7932359576225281, -0.05427015945315361, -0.279459148645401, 1.46322762966156, -1.2818034887313843, 0.30064576864242554, -0.8367266058921814, -0.8535698652267456, 0.49091899394989014, 0.37709128856658936, -1.0055007934570312, 0.116721011698246, -1.1400244235992432, -0.8516146540641785, -1.4738547801971436, 0.27856430411338806, 0.5930877923965454, 0.3482106626033783, -1.6165624856948853, -0.678886890411377, -0.6219040155410767, -1.7738052606582642, -1.0371054410934448, -0.19899685680866241, -0.7976242303848267, -1.7757524251937866, 0.5423811674118042, -0.16477559506893158, 0.41376993060112, 0.7156251668930054, -0.3229834735393524, -1.4255872964859009, 0.5386530160903931, -0.7701792120933533, 1.5311782360076904, -0.2662714123725891, 1.668040156364441, 0.8663477897644043, -1.6262950897216797, 1.435107707977295, 0.6493036150932312, 1.2012887001037598, 0.38810449838638306, 1.8103715181350708, -1.1083197593688965, 0.770484209060669, 2.3872387409210205, 1.9474788904190063, -0.6542513966560364, 0.2889585793018341, -1.2113991975784302, 1.7899541854858398, -1.3180818557739258, 0.3814109265804291, 0.22695614397525787, 1.269466757774353, 1.760201096534729, 0.7773293256759644, -0.4383764863014221, 0.05743732303380966, -0.43549707531929016, -0.15057159960269928, 0.1506982445716858, 2.397315740585327, -1.4806864261627197, 0.6596081256866455, -0.9979169964790344, -0.2088417112827301, 0.052907343953847885, -0.2144150733947754, 1.5935578346252441, 0.4220021069049835, -2.34955096244812, 0.4278750419616699, -0.2319665104150772, 0.8281258344650269, -0.3762587308883667, -0.3843211233615875, -2.0346055030822754, 1.272847056388855, -1.808152437210083, 0.23235806822776794, 0.3608810007572174, 2.0917038917541504, 0.16715335845947266, 1.0139025449752808, -0.9314279556274414, 0.40990617871284485, 1.014886498451233, -1.261610507965088, -0.5547125935554504, 0.6710191369056702, 0.13523291051387787, -1.3286176919937134, 0.2527388334274292, 0.4485766887664795, 0.6495428681373596, -0.4082801342010498, -0.1610630750656128, 1.0504664182662964, 1.5711296796798706, -0.1483639031648636, -1.960359811782837, 0.07430104911327362, -1.1644569635391235, -1.3085919618606567, 0.9635223150253296, -1.869141936302185, 0.08817684650421143, 0.7972330451011658, -0.577082097530365, 0.7712701559066772, 1.3120465278625488, -0.16069653630256653, -0.4974251091480255, -0.5267817378044128, 0.30938994884490967, 0.40896889567375183, 0.1479697972536087, 0.36997145414352417, -0.5588976740837097, 0.6096509099006653, 0.2285853922367096, -0.08792929351329803, -1.372223973274231, -0.3878146708011627, -0.12331187725067139, -0.030775036662817, -1.1133939027786255, 0.7585132718086243, -0.3394434154033661, 0.6417649388313293, -0.2615745961666107, 0.17720578610897064, -0.23327195644378662, -0.8868312239646912, -0.5940729975700378, 0.5466510653495789, -0.7532363533973694, 2.2132885456085205, 0.2949540615081787, -0.765963613986969, -1.7750027179718018, -0.8693243265151978, 1.7229856252670288, 1.173087239265442, -0.7102327942848206, 1.7564252614974976, -1.6017405986785889, 1.4593987464904785, 0.8870904445648193, 1.6347533464431763, -0.5727819204330444, 1.646052598953247, 0.8257561922073364, -1.0979660749435425, -1.1362075805664062, -0.016762688755989075, -0.5623297691345215, -0.7871142625808716, 1.336584448814392, 0.4035756289958954, -0.5870093703269958, -0.2818702161312103, 0.2525775134563446, 1.011505365371704, -2.492475748062134, -0.4844185411930084, 0.3949824571609497, 0.0018682810477912426, -0.950171172618866, -0.001751861535012722, -0.5285845398902893, 0.8370353579521179, 1.351632833480835, 0.8071110248565674, -1.2020189762115479, -0.3054025173187256, 0.9303739070892334, -0.757331371307373, -0.30840590596199036, -1.5537513494491577, -0.5757833123207092, 0.7168747782707214, 0.2710144817829132, -1.3214889764785767, 0.4387162923812866, -0.4746379256248474, -0.07483735680580139, 0.3265952467918396, 1.2730708122253418, 0.1452597975730896, -1.1846293210983276, -2.425809621810913, -1.1291693449020386, -0.9460307955741882, 1.6542210578918457, -0.018184978514909744, -0.20431512594223022, -0.1390434056520462, -1.0829532146453857, -1.0230704545974731, -1.2039204835891724, -0.9735949039459229, 1.1218960285186768, -0.12051623314619064, -0.023295147344470024, -0.888553261756897, 0.12941157817840576, 0.06128857284784317, 0.08972644805908203, 0.06486895680427551, 0.13857901096343994, -0.40524715185165405, 0.3619532585144043, 0.23513570427894592, -1.7483375072479248, 0.30527034401893616, -0.5457658767700195, 0.46671921014785767, -0.6259467005729675, -1.3108986616134644, -0.10814959555864334, -0.44450849294662476, 1.313226580619812, 0.6307185292243958, 1.0709452629089355, 0.5864476561546326, -1.6961899995803833, -0.697634756565094, 0.6519220471382141, 0.27811557054519653, -0.09679360687732697, -2.735065221786499, -1.2798444032669067, 0.7910627126693726, 0.5725610256195068, 0.7102798819541931, -0.28059399127960205, 0.7886140942573547, -1.5124454498291016, -1.259337067604065, -1.012987494468689, -0.0932600125670433, 0.11013593524694443, -0.2892346680164337, -0.1060815304517746, -0.39254072308540344, 1.251927137374878, 0.007630494423210621, -2.1591150760650635, -0.39479583501815796, 2.7666521072387695, 0.005119687877595425, -0.9054659008979797, 0.9204510450363159, -0.50110924243927, -0.6302377581596375, 0.20438151061534882, 0.43549004197120667, -0.18091563880443573, -0.944225549697876, 0.16563618183135986, -0.7325310111045837, -0.8433166742324829, -1.2500503063201904, 1.4403789043426514, 0.78389972448349, 0.11828269809484482, 0.12831833958625793, 0.44847965240478516, 1.146467685699463, 1.2934151887893677, 0.4465274214744568, -1.7242344617843628, 0.17511294782161713, 0.10865747928619385, -0.06742525845766068, -0.32887589931488037, -1.8430116176605225, -0.28949597477912903, -0.21061785519123077, -0.7478491067886353, 0.5379896759986877, -0.38483041524887085, -0.5778470635414124, 0.5483630299568176, 0.6278182864189148, -0.29459959268569946, -0.87655109167099, -0.7149263620376587, 0.10373564064502716, 0.1510985791683197, 0.7462519407272339, -0.09817785769701004, 1.188162922859192, 0.6241670250892639, -1.0214680433273315, 0.22257572412490845, 1.0715258121490479, -0.14750026166439056, -1.6660956144332886, 0.052800148725509644, 0.725603461265564, 0.49607017636299133, -0.27215614914894104, -0.9638349413871765, 0.21481113135814667, 0.446183979511261, 0.4274890124797821, -2.4371564388275146, -0.5276044011116028, -2.0676417350769043, -0.9714087247848511, -0.3729052245616913, 0.6919905543327332, -1.1166141033172607, 0.10505253821611404, 0.8817522525787354, 1.3371578454971313, -1.0402189493179321, 1.1988178491592407, 1.2642141580581665, 0.8798664808273315, 1.016480803489685, -0.14281393587589264, -0.1328073889017105, 1.666904091835022, 0.28492262959480286, -0.31526651978492737, -1.2707210779190063, -0.2497570514678955, 1.782615065574646, 1.4127933979034424, 0.6186875104904175, 0.42771345376968384, -0.1802847534418106, -0.9309176802635193, -0.9001110792160034, 2.1621551513671875, -1.5720129013061523, -2.07730770111084, -0.0964026153087616, -0.021341128274798393, -2.085416078567505, -0.32444024085998535, -0.30705520510673523, 0.5203067064285278, -1.1802046298980713, -0.35868725180625916, 0.4462783634662628, 0.8702563047409058, 0.24331285059452057, -1.975589394569397, 0.029080821201205254, 1.3315526247024536, -0.22659961879253387, -0.539528489112854, 0.23466625809669495, 1.1908996105194092, 0.30545786023139954, 0.9330134391784668, -0.6812182664871216, -0.4216364026069641, 0.06369226425886154, 0.3528279662132263, -0.6090292930603027, -1.6704232692718506, 1.6383016109466553, -0.6509703993797302, -0.6795967817306519, -1.2474143505096436, -2.142716646194458, 0.17724692821502686, -0.986278235912323, 1.0776987075805664, 0.17473796010017395, 2.2612388134002686, 0.6267998218536377, -1.4244451522827148, 0.5663788318634033, -0.30931228399276733, -0.41400206089019775, -1.278649926185608, 0.9152302742004395, 0.8920924067497253, -0.22545619308948517, -1.2836257219314575, 0.0024814491625875235, -0.7807763814926147, 1.248704433441162, 0.3222053647041321, -1.194915533065796, 0.13266314566135406, -1.7600373029708862, 0.9275828003883362, 0.00872985552996397, -0.7670199275016785, 0.7758603096008301, 0.024062639102339745, 1.3768268823623657, -0.0766509473323822, -1.4330533742904663, -0.32017508149147034, 0.36433422565460205, -1.1351028680801392, 0.3531533479690552, -1.4427454471588135, -1.575708270072937, 1.3167617321014404, 0.5692411661148071, 1.237560510635376, -1.1540625095367432, -0.8973967432975769, 0.3056361675262451, -1.7246390581130981, 0.3804195821285248, 0.4478248953819275, -0.8342574834823608, -0.1519099622964859, -0.515923023223877, -0.22996945679187775, -0.5033611059188843, 0.2760390639305115, 0.962212860584259, -1.1675372123718262, 0.12291346490383148, -0.8415931463241577, -1.2092633247375488, 1.1516238451004028, -0.19972774386405945, -0.3982383608818054, 0.5138731598854065, 0.594906210899353, 2.2366387844085693, 0.9256241321563721, 0.5238842368125916, -0.9919747114181519, 0.16629604995250702, -1.3634473085403442, 0.5620774626731873, -1.2205265760421753, -0.09331765025854111, 1.4131169319152832, -1.1689269542694092, 0.20209388434886932, -1.3136777877807617, 0.711432933807373, -0.9673527479171753, -0.9217496514320374, -0.48655441403388977, -0.04829496517777443, -0.7757945656776428, 0.4527522921562195, -0.13695798814296722, 1.1332138776779175, 1.1116302013397217, 0.017443452030420303, 1.5053375959396362, -0.29024451971054077, 0.13209234178066254, 0.22222749888896942, 0.07346843928098679, 1.7475637197494507, 0.6843673586845398, -0.30416232347488403, -0.7271620035171509, 0.10467080771923065, 0.5865974426269531, 1.3939305543899536, -1.1423686742782593, 0.1804729402065277, 0.679816722869873, 1.280340313911438, -0.45591554045677185, 0.2615198493003845, -1.1975202560424805, -0.17270201444625854, -1.2305543422698975, 1.012479305267334, 0.664372980594635, -0.5597289204597473, -0.4523734152317047, -1.5782864093780518, 0.11051422357559204, -0.5701208114624023, -0.17850057780742645, -0.5877602696418762, -0.2207683026790619, -0.9731125831604004, 0.5650341510772705, 0.47821253538131714, -1.6938120126724243, -0.006742438767105341, 1.242039680480957, -0.4260265827178955, -1.2730567455291748, -0.19088026881217957, 0.4818558096885681, -0.08026166260242462, 1.3613669872283936, 2.0930280685424805, -0.5079553127288818, 0.950031042098999, -1.4930604696273804, 1.386657953262329, -1.6400424242019653, -0.08278712630271912, 0.2147475928068161, -1.3208292722702026, -0.8561878204345703, 0.4509873390197754, 0.566432774066925, -1.1268503665924072, -0.8549486398696899, -0.6218986511230469, 2.2281739711761475, 0.6479562520980835, 0.11461120843887329, 0.4788792133331299, -0.40327656269073486, 0.8203652501106262, -0.18881440162658691, -0.18316736817359924, 0.0014752513961866498, 1.0271414518356323, 0.12071073800325394, -0.2390422374010086, 0.668549656867981, 0.37530165910720825, 0.2125389724969864, -0.6987823843955994, 1.1224135160446167, 0.1895761936903, 1.2771821022033691, 0.5192036032676697, -2.7777373790740967, -0.23412363231182098, 1.5353199243545532, -1.0970827341079712, 1.9142303466796875, 0.0274905264377594, 1.2172826528549194, -0.7272810339927673, -0.632288932800293, 0.12664362788200378, -0.21593709290027618, -0.555314302444458, 0.5034447312355042, -0.8444952964782715, -1.102914571762085, -0.4822351932525635, -0.6874405741691589, 0.7207460999488831, -0.1938079297542572, 0.24644982814788818, -1.0040078163146973, -0.1699974536895752, 0.13276736438274384, 0.09702524542808533, -0.3425063490867615, 0.37337929010391235, -0.4248243272304535, -0.6848743557929993, 0.04935053363442421, -0.3959754407405853, 0.6723365187644958, 1.3391141891479492, -0.22047059237957, -1.8594504594802856, -0.808696985244751, 0.1169157549738884, -0.006489957217127085, 1.3909505605697632, -0.21633969247341156, -0.8896066546440125, 0.46958550810813904, 2.404629707336426, -0.7803478240966797, 0.6802257299423218, 0.5106798410415649, -0.5625506639480591, 0.5132483243942261, -0.9875537753105164, -0.692510724067688, -0.18082906305789948, -0.5247639417648315, -0.015506056137382984, -0.9060057401657104, -0.6816748976707458, -0.4690564274787903, -0.07547580450773239, -0.269914835691452, -1.082815408706665, 0.10557247698307037, 1.5258110761642456, -0.7801184058189392, -0.6787469983100891, 0.34079664945602417, 0.9548253417015076, 0.6888130307197571, -0.2534556984901428, 0.31251823902130127, -0.7718111276626587, 0.7604013085365295, -1.0449615716934204, -1.8840997219085693, -0.3936831057071686, 0.3884028196334839, -0.48559150099754333, 0.702238142490387, 0.42409706115722656, -1.8265970945358276, -1.4257030487060547, -0.5998743176460266, -0.5725674033164978, 0.184583842754364, -0.5719908475875854, -0.1763567328453064, -0.759748637676239, 0.4165165424346924, 0.16451051831245422, -1.1266286373138428, 0.8313009142875671, 1.2000333070755005, 0.13880512118339539, -0.09448222815990448, -0.6706397533416748, 0.8174709677696228, -0.055463749915361404, -0.513358473777771, 1.151271104812622, -0.03553887456655502, 1.3934509754180908, 0.15890765190124512, 1.2064038515090942, -2.946436643600464, -0.16475486755371094, 0.27476343512535095, 1.0995415449142456, 0.8559611439704895, 0.46752896904945374, 0.17968334257602692, -1.1787002086639404, 2.684380054473877, 2.0747146606445312, 0.9338122010231018, -1.8992869853973389, 0.5784723162651062, 0.08680430799722672, 0.5105471611022949, 0.07521539181470871, 2.043148994445801, -2.8419322967529297, 0.6682289242744446, -1.7075238227844238, -0.7972655296325684, -0.5088096261024475, -0.24512995779514313, -1.1002777814865112, -1.2963179349899292, -0.7585815787315369, -1.8429454565048218, 0.6307123899459839, -0.4025503098964691, -0.8939015865325928, 0.6496796607971191, -0.28123578429222107, -0.7790305018424988, 2.2008235454559326, 1.3352400064468384, -1.6286780834197998, 0.5719298720359802, 0.8507750034332275, 0.7848127484321594, -0.5014781951904297, -0.7468158602714539, -1.6510390043258667, -0.8421458005905151, -0.5008867383003235, 0.335308313369751, 0.8881882429122925, 0.11849990487098694, -0.609586775302887, 1.0792826414108276, -0.6394321918487549, -0.5232906341552734, 0.2482522577047348, -0.23661909997463226, -0.7608873248100281, -0.05294373258948326, 0.3040206730365753, 2.4539859294891357, -0.21343496441841125, -0.15955734252929688, -0.09689027816057205, 0.2731637954711914, 0.7262867093086243, 0.49783676862716675, 1.0750410556793213, -0.06544869393110275, -0.2740258574485779, 0.6901294589042664, 0.8477187156677246, 0.11984992772340775, -0.349850058555603, 0.26870888471603394, 0.12033326923847198, 0.07693320512771606, 1.5124123096466064, -0.594157874584198, -0.20145493745803833, 1.2502235174179077, 2.7414064407348633, 0.6083289384841919, -0.34005892276763916, 0.7132807970046997, -0.3926401734352112, -0.3337893486022949, 0.00018934677063953131, 1.1275769472122192, -0.17683307826519012, 1.1098623275756836, -1.6260679960250854, -0.7869147062301636, 0.33922016620635986, -0.43338146805763245, -1.2969260215759277, 0.7538247108459473, 0.3476587235927582, -1.2206884622573853, -0.6345636248588562, -0.9540112614631653, -1.1752477884292603, 0.3915267288684845, -0.5819841623306274, 0.40290799736976624, -0.20200534164905548, -0.807673990726471, -1.1762800216674805, -0.08488902449607849, 0.7329593896865845, 0.1531769186258316, -0.16759279370307922, -0.49691691994667053, -0.5974236726760864, 2.9078433513641357, 0.21006783843040466, 0.5910272598266602, 1.1764081716537476, -0.9174330234527588, 0.5238046050071716, -0.42847001552581787, 0.8157298564910889, -0.43287283182144165, -0.7296212911605835, 1.100726842880249, -0.7908993363380432, 0.15063969790935516, -0.386545330286026, -0.3204842209815979, 0.05378378927707672, 1.1602274179458618, -1.5743554830551147, -1.1882725954055786, 0.18257997930049896, 0.5316405892372131, -0.5295959711074829, -1.4950134754180908, -0.746044397354126, -1.0006636381149292, -0.2932051122188568, -0.08295124024152756, -0.17072072625160217, -0.3470366895198822, -0.317947119474411, -1.0015959739685059, -0.8142666816711426, 1.0244280099868774, 0.288887083530426, -0.19306784868240356, -0.6631722450256348, -0.45804035663604736, 2.1575021743774414, -0.8355700969696045, 0.09421442449092865, -0.6904245615005493, -1.2512911558151245, 0.4950750470161438, 0.6844432353973389, 1.2940131425857544, -2.3076751232147217, -1.8731330633163452, 0.14389163255691528, 1.3978960514068604, 0.2877938449382782, 0.06485439836978912, 1.2596622705459595, 1.8352442979812622, -1.8985838890075684, -1.2508430480957031, -0.0031532561406493187, 0.29889312386512756, -0.12235081195831299, -0.7528824806213379, 0.8972018957138062, 1.1697006225585938, -0.7431155443191528, -1.336380124092102, -1.556862235069275, -1.3007265329360962, 1.2831820249557495, 0.46447357535362244, 0.4934847354888916, 0.2742043435573578, -0.9697377681732178, -0.3730335831642151, -1.3097763061523438, -0.7942527532577515, 0.4909072518348694, -1.1834924221038818, -0.813847005367279, 0.7022089958190918, -1.5642297267913818, -1.0639142990112305, 0.2883830964565277, -0.7789463996887207, 0.6898567080497742, -1.7966974973678589, -0.1646706908941269, -1.56168532371521, -0.5376260280609131, 1.231378197669983, 0.4892350137233734, -2.2490015029907227, 0.8788133263587952, 0.5689085721969604, -0.026893658563494682, -0.550451934337616, -1.2871431112289429, -2.064389944076538, 1.8877601623535156, 0.8343064785003662, 1.2310177087783813, -0.26663392782211304, -0.21540938317775726, 1.515417218208313, 0.15375296771526337, 0.8275209069252014, -1.86416757106781, 1.5739634037017822, -1.0474214553833008, 0.9304982423782349, -0.976036548614502, 1.4653855562210083, -0.6433140635490417, 0.9332540035247803, -0.7455182075500488, 0.10146661102771759, 0.3837825655937195, 0.21779794991016388, 0.21627482771873474, 1.5089446306228638, 0.6793423295021057, 0.8527617454528809, -1.1233501434326172, 0.03860969468951225, -0.38322263956069946, 0.598619818687439, 0.25149741768836975, -1.2264806032180786, -2.5358963012695312, 1.1703752279281616, -1.3571442365646362, 0.07505148649215698, 0.6647471189498901, -0.9678993225097656, 0.11311004310846329, 0.6732650399208069, -1.3966509103775024, -1.7992435693740845, 0.6471050381660461, 0.9550793170928955, 1.4868298768997192, 0.8772163987159729, -0.43750762939453125, -0.9616132974624634, 0.7332254648208618, 0.46308067440986633, 0.5154411196708679, -1.8281463384628296, -1.8065009117126465, 0.9003488421440125, -0.9900946617126465, 0.6753899455070496, 0.47016873955726624, 0.5036776661872864, -0.6407189965248108, 1.661354899406433, -1.7377768754959106, -0.08997759968042374, -0.3283514976501465, -0.11360032111406326, 0.2423853576183319, 0.07311974465847015, -0.47327518463134766, 0.17494159936904907, 0.20106950402259827, 0.4792594015598297, -0.6695821285247803, -1.4386972188949585, 1.1909255981445312, -0.38466355204582214, 0.5978264212608337, 0.004382973536849022, -0.9421094655990601, -0.43086302280426025, 0.3900977373123169, 0.11446664482355118, 1.3101674318313599, 1.5008230209350586, 1.1326003074645996, -0.04414856433868408, 0.7093416452407837, -1.1513665914535522, -0.9284560680389404, 0.15788395702838898, -0.4615231454372406, -0.9333658218383789, 0.05271553248167038, -0.7337960004806519, 0.9220949411392212, 1.1642353534698486, 0.9171423316001892, 2.0637593269348145, -0.5620996952056885, -1.404854416847229, -0.06604728102684021, 0.639446496963501, -0.8378183245658875, -0.309883713722229, -0.4549408555030823, -0.5064160823822021, 0.7312121391296387, -0.8723170161247253, 1.93485689163208, -0.9920010566711426, -0.37436622381210327, 0.1371670961380005, -0.6286607384681702, 0.2900192141532898, 1.26718008518219, 1.9972892999649048, 2.01265025138855, -0.8307967782020569, 1.2079044580459595, -0.1537858247756958], [0.8636983036994934, 0.34806883335113525, 0.7109518647193909, -0.48791417479515076, -0.5477650165557861, 0.19186922907829285, -2.112349271774292, -0.5489882230758667, -0.5006207823753357, 0.20170971751213074, 0.36801084876060486, 0.14451473951339722, -1.2020502090454102, -0.04412829875946045, 1.3545348644256592, 3.3675146102905273, 0.1813281774520874, 1.4485324621200562, 1.268945574760437, -0.16930240392684937, -0.6337098479270935, 0.28232598304748535, -0.760949432849884, -0.6913809180259705, 0.9261082410812378, -0.8206896781921387, 0.28167524933815, 2.0338540077209473, -3.489447593688965, 0.2561360001564026, -0.310133159160614, 3.078228235244751, 0.6772581934928894, -0.36287885904312134, -0.33846402168273926, -0.3470219075679779, -0.9497830867767334, -0.13163720071315765, -0.8385931253433228, 0.4771067202091217, 0.19903351366519928, -0.11015535891056061, -0.31414365768432617, 0.5364872217178345, 0.9606888294219971, 0.04158983752131462, 0.6061773300170898, 0.7045304179191589, -0.8320422768592834, -0.4731864333152771, -0.3936406970024109, -1.6955195665359497, -1.1319711208343506, 0.23545187711715698, 1.5244145393371582, -0.1826811581850052, -0.053224194794893265, -0.25855550169944763, -0.2856096029281616, 0.4489353895187378, 1.7388582229614258, 0.8642486929893494, -0.37423303723335266, 0.5994893908500671, -0.1197502389550209, -0.09092196822166443, -0.861833930015564, 0.039939384907484055, 0.7250742316246033, -0.2723110318183899, 0.3684973120689392, 0.12883156538009644, 0.5543074011802673, -0.0722149908542633, 2.5407090187072754, 0.15793867409229279, -0.6990230679512024, 1.1682060956954956, -1.2468844652175903, -0.24098548293113708, 0.825350821018219, 0.9510684609413147, -0.22345836460590363, 1.4387346506118774, -0.3628388047218323, -0.12678930163383484, -1.4648619890213013, 2.043142318725586, 1.166717529296875, 1.2689515352249146, 0.7078614234924316, 0.6892747282981873, -1.310592532157898, 1.4922730922698975, -0.18746346235275269, 0.11043412238359451, -0.8389273285865784, 1.259310007095337, -0.05102818086743355, -0.8656852841377258, 0.11544385552406311, 1.0838768482208252, 0.6241762638092041, 1.4333347082138062, -0.1348535120487213, -0.29909104108810425, 1.2065552473068237, -0.5226126313209534, 1.3339009284973145, -0.6809068918228149, -0.23707452416419983, 0.5812290906906128, 1.3849157094955444, 0.9645267128944397, 0.7136551737785339, 0.4213305413722992, -2.577404499053955, 1.688205361366272, -0.2608675956726074, -0.9161118268966675, -0.30062881112098694, 0.4306167662143707, -3.2198925018310547, -0.7147819399833679, 1.0580682754516602, -1.1059033870697021, 1.4002083539962769, 0.6877354383468628, -1.1226844787597656, 0.03351980820298195, 0.004758814349770546, -0.20380418002605438, -0.4915297031402588, -0.15622703731060028, 0.06551147252321243, -0.15939435362815857, 0.861687421798706, -0.1810758262872696, -0.39093920588493347, -3.6472549438476562, 0.16310414671897888, 1.5643113851547241, 0.48572778701782227, 0.8257517218589783, -0.9671967029571533, 0.32701724767684937, -0.39445415139198303, 0.10559574514627457, 1.795749306678772, -0.6350041031837463, -0.44456854462623596, -0.7379075884819031, -0.7696360945701599, 0.794701337814331, 1.8315287828445435, 1.5348231792449951, -0.10216197371482849, -1.3990778923034668, -0.812476634979248, -0.422037810087204, 1.0814708471298218, 0.11615359038114548, -0.3791155219078064, -1.1708344221115112, -0.5280579328536987, -0.8608127236366272, 0.3452005684375763, 0.15357176959514618, -0.07042352855205536, -0.044858697801828384, 0.6497666239738464, -0.2863173484802246, 1.569675326347351, -0.6418404579162598, -1.1133449077606201, -0.4458965063095093, -0.39131081104278564, -0.7822393178939819, -0.9252145886421204, -1.696008563041687, -0.005403589457273483, -0.9912338852882385, 1.5800834894180298, 0.34085381031036377, -1.034045934677124, 0.2951284348964691, -1.0803987979888916, -0.032370392233133316, 0.3293406069278717, -0.9290141463279724, 0.0391918309032917, 2.0433177947998047, 0.4269946813583374, 0.22429171204566956, -0.3078899085521698, -0.316616028547287, -2.0692787170410156, -0.21371638774871826, 0.6399940252304077, -0.10878215730190277, -1.8853334188461304, 0.09080668538808823, -0.16499026119709015, -1.7257622480392456, -0.0571451336145401, 0.11042004078626633, 0.3907843232154846, 0.15939010679721832, -0.4371371865272522, -0.29710516333580017, 0.8712652325630188, -0.4921497702598572, 0.6929623484611511, -1.1017543077468872, -1.9721381664276123, -0.25228768587112427, -0.0020119778346270323, -0.7451016306877136, -1.4164286851882935, -0.9209725856781006, -1.1065095663070679, 1.1984325647354126, -0.5034423470497131, -0.2203308343887329, -0.7062854170799255, 0.3158714473247528, 0.02963927946984768, -0.4510115683078766, -0.48839622735977173, 0.0009591708658263087, 0.5604773163795471, -0.7853736877441406, -0.49213916063308716, -1.4499740600585938, 1.0538504123687744, -0.3114202618598938, -0.07340753078460693, -0.8320236206054688, -0.4277188777923584, -0.14113087952136993, 0.39619138836860657, -0.09719029814004898, -1.691668152809143, -1.2377755641937256, 0.29393839836120605, -1.6633360385894775, -0.434141606092453, -0.24910679459571838, -0.5337876677513123, -0.36925309896469116, -0.6324402689933777, -1.1715623140335083, -0.3796091079711914, -0.9923913478851318, -0.41619035601615906, -0.8208878040313721, 2.8156116008758545, 1.4216532707214355, 0.8341050744056702, 0.7207787036895752, -1.5431442260742188, 0.7221477031707764, -0.1673223078250885, -1.502457618713379, -0.9887952208518982, -0.011789976619184017, -0.5952521562576294, -1.3905212879180908, -0.46757203340530396, 2.643137216567993, 1.0754741430282593, -0.25743764638900757, 1.9037765264511108, -0.6269147396087646, -0.5171303153038025, -1.0863231420516968, -0.15488529205322266, -0.13965460658073425, -0.6552395224571228, 0.5897195935249329, -1.2193089723587036, 0.4044942557811737, 1.2985305786132812, -0.1528513878583908, 1.1726224422454834, -1.0467529296875, -0.4470423460006714, -0.5927262306213379, -0.04231392592191696, 1.6507453918457031, 0.6216999888420105, -0.0005216478602960706, 1.5877779722213745, 1.3482961654663086, 0.08152427524328232, 0.12128910422325134, -1.1205401420593262, -0.6980442404747009, 2.7965288162231445, 1.3417119979858398, 0.18000085651874542, 0.5741002559661865, 0.4911903738975525, 0.9399840235710144, 0.7116674780845642, 0.14631876349449158, 1.2377289533615112, -1.4079128503799438, 0.2717376947402954, -1.2933939695358276, 1.3508473634719849, 1.3202000856399536, -0.9014587998390198, 0.051289331167936325, -0.4082924425601959, 0.4548414349555969, 0.47987794876098633, 1.0748814344406128, -0.6260814070701599, -0.8682908415794373, -0.15852154791355133, 0.9402419924736023, 0.6169494986534119, 0.6450253129005432, -0.7895193696022034, -0.4002003073692322, -1.3198230266571045, 0.2783152163028717, -1.015033483505249, 0.6597822308540344, -1.91989266872406, 0.16055989265441895, -0.5130753517150879, -0.8085921406745911, 0.7729977965354919, 2.597651243209839, -0.05612899735569954, 0.2052549421787262, -0.9557214975357056, -1.0686982870101929, 0.43416258692741394, -1.599944829940796, 0.5867288708686829, 1.083181381225586, 0.6946629881858826, 0.264684796333313, 1.2944691181182861, 0.9273064136505127, 0.8403519988059998, -1.9555673599243164, -1.7481601238250732, -0.33822929859161377, -1.632439374923706, -0.18349863588809967, -1.7984263896942139, 0.8094241619110107, -0.628261148929596, -0.8332016468048096, 1.2423945665359497, 0.29520976543426514, -0.1377982795238495, -0.04102984443306923, 0.33254778385162354, 0.9170883893966675, 0.12151360511779785, -0.5511661767959595, 0.27997323870658875, -0.012430090457201004, -0.488662451505661, -0.2755562961101532, -0.3630005717277527, -0.5725158452987671, 1.0754284858703613, -0.7326232194900513, 1.5522830486297607, -0.47529110312461853, 1.561821460723877, -2.133937120437622, 0.6314070224761963, 0.19320940971374512, -1.0488637685775757, 1.2308367490768433, -0.3717420697212219, 0.770007848739624, -0.8538554906845093, -0.739409863948822, 0.9262232184410095, 0.9227207899093628, -0.15280528366565704, 0.08045092970132828, 1.2332029342651367, 0.7524365782737732, 1.360761046409607, -0.5766206383705139, -0.08378376066684723, 0.24835391342639923, 0.36630651354789734, 0.1349305659532547, 0.7497923374176025, -0.3946319818496704, -0.033983293920755386, -2.2407689094543457, -0.6068379282951355, -1.089491605758667, 1.8822633028030396, 0.3720657229423523, 1.1508617401123047, 0.3013775646686554, -0.37121137976646423, 0.10591115802526474, -0.010834657587110996, 1.4062258005142212, 0.2657322287559509, 1.8895927667617798, -1.5487974882125854, 0.4892353415489197, 1.1823761463165283, -0.4184035658836365, 0.03469272330403328, 1.105255126953125, 1.3049135208129883, -1.2817747592926025, 0.3064424991607666, 1.3664950132369995, -0.14906150102615356, -0.3195338547229767, -0.4542720317840576, -0.10954420268535614, -0.7229944467544556, -0.5348427295684814, -0.6435509920120239, 0.38558119535446167, -1.2904200553894043, 1.2425864934921265, 1.0104848146438599, 0.6733043789863586, -0.10981234908103943, 1.2583261728286743, 0.14426584541797638, -0.729265034198761, -1.441536545753479, 0.17830978333950043, -0.8352141976356506, -1.1350560188293457, -1.1366097927093506, 0.16220857203006744, -0.5927267670631409, -1.9990324974060059, -1.2104600667953491, -0.43424180150032043, -0.12134111672639847, 0.31438735127449036, 0.7315671443939209, 1.5351427793502808, -1.1401267051696777, 1.8298137187957764, 0.060498155653476715, -0.8856427073478699, 1.1325186491012573, 0.08740904182195663, -1.042636752128601, -1.2600195407867432, -0.10343316942453384, 0.3708142042160034, 0.9268051981925964, -1.6271873712539673, -0.1570778787136078, -2.0333333015441895, 1.7170113325119019, 0.8434619903564453, -0.2804996371269226, 0.6183558702468872, -0.44808459281921387, 0.5128670930862427, 0.10381914675235748, 1.5818376541137695, -0.45764732360839844, -0.571938157081604, 0.4920097887516022, 1.2156119346618652, 0.35524943470954895, -0.844316840171814, -0.8259279131889343, -0.654244065284729, -0.3161008059978485, -1.1609424352645874, -0.32742902636528015, 0.041154731065034866, 0.12335900962352753, -1.6619651317596436, 0.11369125545024872, -1.0214534997940063, -1.1726584434509277, 0.12037791311740875, -0.6911599040031433, 0.18666090071201324, 0.4583948850631714, -1.7466009855270386, 0.12548509240150452, -0.9598727226257324, -2.3163042068481445, 0.7364609241485596, 0.622467041015625, -1.344261884689331, 1.586137294769287, 0.3429405987262726, 1.3576033115386963, 0.48464637994766235, -0.54441237449646, -0.21369382739067078, -0.9186119437217712, 0.05659826844930649, 1.056252121925354, 0.21506233513355255, -0.18097376823425293, -2.097994089126587, -1.1784698963165283, -1.897138237953186, 0.756117582321167, -0.024756385013461113, 0.895776629447937, 1.1174379587173462, -0.34982830286026, 1.010095477104187, 0.7588499784469604, -0.340488463640213, -1.4432045221328735, 0.18449801206588745, 0.47958770394325256, -0.3543527126312256, 0.2613838315010071, 0.4448402523994446, -0.5519766211509705, 0.906315803527832, 0.22144533693790436, 0.3583524227142334, 1.0728729963302612, -0.25808271765708923, -0.47907117009162903, 1.7307277917861938, -1.1362767219543457, -1.1979427337646484, 1.174259901046753, -0.717544436454773, 0.34943997859954834, -1.2213791608810425, -0.1571330428123474, 0.7140769958496094, 0.0612008236348629, 1.5548557043075562, 1.7469350099563599, -0.07093637436628342, -0.8714661598205566, 0.44504228234291077, 0.32876214385032654, 1.2883411645889282, 0.41494444012641907, -0.45810097455978394, -1.1477460861206055, -0.45302239060401917, 0.9923490881919861, 1.4415284395217896, -0.1933402568101883, 1.5594967603683472, 1.5689572095870972, 1.8315249681472778, -0.9473904967308044, 0.4102908670902252, -0.3264334797859192, 0.9102014899253845, -1.5701758861541748, 3.0568816661834717, -0.46368467807769775, -1.2018722295761108, -0.7309791445732117, 0.7466320991516113, 0.670312225818634, 0.31056034564971924, -0.2034536600112915, -1.2422173023223877, -1.376323938369751, -0.7168503999710083, -0.318785697221756, -1.3466628789901733, 0.228581503033638, -0.9804322719573975, -0.6844785809516907, 0.7718459963798523, -0.5063597559928894, -1.8658589124679565, 0.034882184118032455, 0.4696727395057678, 0.2899665832519531, 0.21879461407661438, -0.9744936227798462, 0.7810373902320862, -0.20652765035629272, 0.41239306330680847, 0.021671392023563385, -2.3060338497161865, 0.2997392416000366, 0.3506722152233124, -0.25443434715270996, -0.0932089164853096, -0.04413137584924698, 1.0120422840118408, 0.38975661993026733, -0.5837309956550598, -2.0830461978912354, 1.3994311094284058, 1.3504638671875, 2.0143089294433594, -1.42892587184906, 1.659047245979309, -0.20313282310962677, 0.1562698632478714, 0.8408730626106262, 0.35890766978263855, 0.20223574340343475, -0.06121641770005226, 1.2880243062973022, -1.3360720872879028, 2.698347568511963, -0.8863875269889832, -0.3461434841156006, -0.9402803778648376, -0.042565714567899704, 1.7471661567687988, -0.9148697257041931, 0.20277182757854462, -0.09159896522760391, 0.5389484167098999, -0.9094921350479126, 0.08175314217805862, -0.9724065065383911, 1.1653999090194702, -1.1092671155929565, -0.2996547520160675, -0.1383216828107834, 1.615072250366211, -0.8244064450263977, -0.5493712425231934, -0.02046504057943821, 0.5079418420791626, -0.5941447019577026, 1.6253595352172852, 0.885474443435669, -1.2062867879867554, 0.3245542347431183, -0.13170185685157776, 2.2226974964141846, 0.17570027709007263, -0.0636376440525055, 0.7327624559402466, 0.15775026381015778, 1.7687374353408813, 1.9994562864303589, -0.5577452182769775, -1.3399981260299683, -0.1373090147972107, 0.0028769955970346928, 0.040537279099226, -2.7757816314697266, -1.5235079526901245, 0.07878243923187256, 1.1114161014556885, -1.766441822052002, -0.6415172815322876, 0.5835599899291992, 0.9046124219894409, -0.3579866290092468, 1.540083885192871, -0.1551695317029953, -0.29665571451187134, -0.34467291831970215, 0.09969407320022583, -1.99815833568573, -0.0531369149684906, 0.18526902794837952, -1.5712828636169434, -0.4681653678417206, -0.9840528964996338, -0.20774325728416443, 2.6438698768615723, -1.4326186180114746, 0.531887412071228, 0.4204268455505371, 0.7271364331245422, -1.3270015716552734, 0.40195906162261963, 0.252348393201828, -0.1326538324356079, -0.08650540560483932, 1.5941784381866455, -1.0600587129592896, -0.04361405596137047, 0.052180130034685135, -0.4730994403362274, 2.6049211025238037, 1.0509672164916992, -1.5210596323013306, -0.030351264402270317, 1.5311120748519897, -0.5123089551925659, 1.9792053699493408, -2.3291821479797363, -0.7052884101867676, -1.2793874740600586, 0.7821357846260071, 1.956613540649414, -1.5817210674285889, -1.059518575668335, 0.1397823691368103, 1.7323402166366577, 0.47290730476379395, 0.2981996536254883, 0.9099888205528259, 0.5138245224952698, -0.4468148648738861, -0.47654837369918823, -0.9336091876029968, -0.29022952914237976, -0.6306832432746887, 0.4674581289291382, 0.3208959400653839, -0.2220125049352646, -0.6598851084709167, -0.8573223948478699, 2.6462886333465576, 0.37144413590431213, -0.2269313633441925, -1.6580779552459717, -1.292761206626892, -1.641180396080017, -0.5732962489128113, -1.7138792276382446, 1.046034336090088, -1.2032647132873535, -1.9011549949645996, -2.430600643157959, 1.6082438230514526, 1.504778265953064, -0.6509280204772949, -0.7645772099494934, -0.05511556565761566, -1.9289169311523438, -1.2642836570739746, 2.451686382293701, 0.02954593487083912, -0.6010913252830505, 0.8174893260002136, -1.219026803970337, 0.012490418739616871, 1.6336719989776611, 0.7417641878128052, -1.2552143335342407, 1.162672996520996, -0.43647658824920654, -0.6493368148803711, 1.5299350023269653, 1.8946352005004883, -0.28166478872299194, -0.8724033236503601, 1.250467300415039, 0.3496495485305786, -0.17406341433525085, -0.6745869517326355, -0.5819483399391174, -0.9265594482421875, -0.9901382327079773, -0.08215842396020889, -0.4017011225223541, 1.179785132408142, 0.23190905153751373, -0.23841874301433563, -1.4965038299560547, 0.4338826835155487, -0.9596572518348694, 0.46329402923583984, 0.481841504573822, 0.5401431918144226, -0.20000015199184418, 0.0399029366672039, 2.962592363357544, -1.5238703489303589, 1.5914427042007446, 0.22159095108509064, 0.6116462349891663, -1.271794080734253, 2.0358786582946777, 1.1622967720031738, -0.23476235568523407, -0.01307603344321251, 0.5561054944992065, 2.4557063579559326, 0.21254567801952362, -0.747860312461853, 0.23286157846450806, -1.2527687549591064, -0.19951649010181427, 0.14786134660243988, 0.4550788998603821, -3.2621076107025146, 0.3116820156574249, -0.8324885964393616, 1.5909334421157837, -0.4275749623775482, -0.6239374876022339, 0.9648014903068542, -0.9883965253829956, 0.26647618412971497, -0.8839123845100403, 0.07396896183490753, 0.7506695985794067, -1.1879022121429443, -1.3190736770629883, -0.7097423076629639, -0.15363144874572754, 0.004535978194326162, 0.05389605462551117, -1.6112239360809326, 0.48816171288490295, -0.9939712285995483, -0.39007315039634705, 0.3721500039100647, 0.5885105133056641, 0.9303917288780212, -0.9715012311935425, 1.19511079788208, 2.945058584213257, 0.6580834984779358, -1.555442214012146, -1.2810486555099487, 0.5282636284828186, 0.7118436694145203, 0.07296730577945709, -0.062432076781988144, 2.285043478012085, 1.0191148519515991, 0.23584552109241486, 0.006756193935871124, 0.12423799186944962, -1.9391791820526123, 0.1602393090724945, -0.02671801671385765, -0.049090202897787094, 1.1648226976394653, -1.140689730644226, -0.9440503716468811, -1.0333563089370728, -0.01840321160852909, 1.3518695831298828, 0.37722277641296387, 0.4272973835468292, -1.2846710681915283, -0.348553329706192, 0.2799803912639618, -0.12505434453487396, 1.4747401475906372, -0.6895010471343994, -1.4429315328598022, -0.6416409015655518, -0.1751195639371872, 1.0238312482833862, -0.18974025547504425, -2.104048490524292, -0.06710658967494965, 1.0282812118530273, 1.5662857294082642, 0.7098363637924194, -2.0111987590789795, 0.6050387620925903, 0.1556229144334793, 1.4565223455429077, 0.3876332938671112, 0.49194884300231934, 0.7347326874732971, 1.4737060070037842, -0.23865781724452972, -0.019726227968931198, -1.1546298265457153, -1.563215732574463, -1.1513628959655762, 1.3963887691497803, 0.5584431290626526, 0.333649218082428, -0.680941641330719, -0.3102286458015442, 1.8831013441085815, -0.36680200695991516, -0.3749353289604187, 0.5181030631065369, -0.12008987367153168, 1.0758044719696045, 0.8924233913421631, -0.6441783308982849, -1.598974347114563, -0.0828319638967514, -0.6234723329544067, -0.19804176688194275, -0.37749311327934265, -1.1517373323440552, -2.042625665664673, 2.0664961338043213, 0.45391902327537537, -2.0245697498321533, -1.921146035194397, -0.4903988838195801, -1.1687068939208984, 0.710754930973053, -0.816440761089325, 0.3681061565876007, 0.0601002499461174, -1.0587185621261597, 1.5004247426986694, 0.058252956718206406, -1.036057472229004, 0.3088456690311432, 0.22206684947013855, -0.8815136551856995, 2.5096189975738525, -0.5305169820785522, 1.1921355724334717, 1.1056183576583862, -0.7058113813400269, 0.7666719555854797, 0.6175263524055481, 0.3827176094055176, 0.03434529900550842, 0.18945559859275818, -1.3942689895629883, 1.6913371086120605, -0.7894642353057861, 0.13528825342655182, -0.156251922249794, -0.10543053597211838, 0.6432631015777588, 0.3732374608516693, -0.22637054324150085, -0.7449182271957397, -0.31778204441070557, -0.05196161940693855, 1.2084718942642212, 0.6209691166877747, 1.1099966764450073, -2.054443597793579, 0.3505166172981262, -0.8439357280731201, -0.37076786160469055, -0.59658282995224, 0.03952514007687569, 0.1620776653289795, -0.42659759521484375, -1.832857608795166, 0.35020315647125244, -0.06687939167022705, 0.9631470441818237, 0.1160544827580452, -2.0497210025787354, -0.7255732417106628, 0.919482409954071, 0.779571533203125, 1.1691346168518066, 1.1920486688613892, 0.3569319546222687, -0.9224544763565063, 0.591669499874115, -0.10596168786287308, -0.9212055802345276, 0.7147144079208374, 1.5507742166519165, -0.3440946340560913, 0.7719906568527222, 0.7029026746749878, 0.2654888927936554, 0.7913439273834229, 0.29789599776268005, 0.06815812736749649, 0.4275279641151428, -0.6360867619514465, 2.7153995037078857, 0.5781679153442383, -0.32683324813842773, 0.4497695863246918, 2.0038836002349854, -1.0004215240478516, 1.4650204181671143, 0.15301765501499176, -0.7769197821617126, -0.2718288004398346, -1.1227182149887085, 1.257874608039856, -0.8575146198272705, -2.9122724533081055, -1.449598789215088, -0.9368001818656921, 0.09632070362567902, -1.0112667083740234, -2.387396812438965, 0.40949252247810364]], "dtype": "float32"}, 0], "e": {"__nd__": [0.1303662359714508, 7.942448139190674, 3.1318070888519287, 0.1495051234960556, 2.0223140716552734, -3.118762493133545, 4.213118076324463, 5.865973472595215, 3.219404458999634, -7.14301872253418, 1.311577558517456, 3.22459077835083, 1.2874574661254883, 4.01745080947876, 12.899164199829102, 10.467720031738281, -4.072917461395264, 7.1753830909729, -8.4945068359375, 0.2282472401857376, 8.936738014221191, 13.164607048034668, -3.7949023246765137, -3.3575315475463867, 0.033204492181539536, -1.1716688871383667, 8.603715896606445, 2.1475393772125244, -6.434089660644531, 17.387508392333984, -2.382859230041504, 12.594056129455566, -0.5404278039932251, 3.7024483680725098, -9.419950485229492, 1.277662992477417, -3.0261027812957764, -0.025407761335372925, -1.035825252532959, -1.203620433807373, -10.928888320922852, -10.120882987976074, -1.6678035259246826, -10.083747863769531, 2.7360312938690186, -4.869149684906006, 1.5504924058914185, 5.023166179656982, -8.419169425964355, -4.429501056671143, 0.6673430800437927, 5.313868045806885, -4.790440082550049, -6.863061904907227, 6.774737358093262, 9.163220405578613, -0.5987510085105896, -12.6219482421875, 7.127126216888428, 1.651593565940857, 1.4476475715637207, 4.481375217437744, 9.01885986328125, 4.0437703132629395, -12.467351913452148, 1.1485577821731567, 7.197064399719238, 0.016383599489927292, -5.019540309906006, -1.4688831567764282, -7.233071327209473, 2.460343360900879, -3.309445381164551, 8.792203903198242, 8.067123413085938, 2.8489670753479004, -4.017659664154053, 7.7316365242004395, -9.63280963897705, 0.733483076095581, 3.82639741897583, 6.500014781951904, -1.1747100353240967, 6.409908771514893, -7.283446788787842, 1.6213726997375488, -0.9665752649307251, 6.592803955078125, -1.096334457397461, 7.406190872192383, 5.623939037322998, 6.414428234100342, 7.451794624328613, -9.946892738342285, -1.3636468648910522, -4.510009765625, 6.922236442565918, -6.790212631225586, -5.818358898162842, 2.7505381107330322, -3.342419385910034, -3.9534542560577393, 6.6305108070373535, -3.824899196624756, -6.541652202606201, -3.216156244277954, 3.220771551132202, -0.11359622329473495, -1.7098990678787231, 12.325471878051758, -12.682861328125, -4.580995559692383, -3.4326701164245605, 3.957996368408203, 2.258323907852173, 7.705092906951904, 5.536435127258301, -7.779197692871094, 1.0859501361846924, 3.068845748901367, -4.913083076477051, 2.006927251815796, -5.96091365814209, -3.543470859527588, 12.542159080505371, 7.17027473449707, -2.5764334201812744, 2.1533076763153076, -1.5622057914733887, -4.417745113372803, 9.105730056762695, -5.123063564300537, 2.551542043685913, 2.5686564445495605, 6.5287981033325195, 3.374519109725952, 3.234053373336792, -4.083396911621094, 4.185503005981445, -3.5892598628997803, 8.3275785446167, 0.5091716647148132, 10.143301963806152, 1.7902889251708984, -6.957454681396484, -4.97188663482666, 3.987706184387207, 4.642353057861328, 7.255066394805908, 11.2075834274292, -7.62013578414917, 3.0286061763763428, 6.8224406242370605, 0.11033608764410019, -6.787503242492676, -5.307346820831299, -5.394465446472168, 7.8531904220581055, -7.227862358093262, -0.5941049456596375, -1.797646164894104, -5.210943222045898, -4.991922855377197, -2.1316702365875244, -0.08084835857152939, -3.7162511348724365, -7.252719879150391, -8.867491722106934, -1.4617335796356201, 8.024399757385254, 12.706844329833984, 0.5951134562492371, 3.256136178970337, -5.077458381652832, -2.7420363426208496, 7.494511127471924, -1.168220043182373, 4.65544319152832, -2.505836009979248, -5.2759318351745605, 4.4465484619140625, 1.9526665210723877, -1.6925501823425293, 4.906536102294922, 2.166548252105713, 5.949892044067383, -1.5189757347106934, -4.642982006072998, 0.744806170463562, -19.461315155029297, -7.955129146575928, 8.911636352539062, 2.772759199142456, -3.5700936317443848, -1.3406000137329102, -5.2719879150390625, -1.3208779096603394, -0.8285627365112305, -8.695207595825195, -1.834043025970459, -2.235058307647705, 2.5158402919769287, -4.584742069244385, -4.702115058898926, 3.198897123336792, 0.42461127042770386, 3.240896463394165, -9.475093841552734, -4.865116596221924, -3.828834056854248, -1.6793043613433838, -1.6436501741409302, 5.135335445404053, -8.836287498474121, -5.212972164154053, -6.131745338439941, 4.821865081787109, 5.1576247215271, 0.6473215222358704, -6.4511494636535645, -4.4401116371154785, 19.57959747314453, -6.446531772613525, 3.7833688259124756, -4.773352146148682, -5.572037220001221, -3.902320384979248, 0.07520432770252228, 4.46151876449585, -2.1805293560028076, 3.343355178833008, 4.2841410636901855, -6.7532501220703125, -2.777278423309326, 6.379439830780029, -2.2368054389953613, 1.393874168395996, -3.228210210800171, -2.426187515258789, 6.754514217376709, 9.75088119506836, -7.705393314361572, -0.7243124842643738, -7.161100387573242, 1.4728842973709106, -8.364093780517578, -13.137147903442383, -2.4620578289031982, -12.30873966217041, -1.5550541877746582, 4.456677436828613, 1.9647324085235596, -1.7110364437103271, -1.8461226224899292, 0.7397051453590393, 0.07485470920801163, -6.765136241912842, -6.7409348487854, 2.3205151557922363, 4.072880744934082, -5.297547817230225, 0.8380939960479736, -1.2983916997909546, -1.2757833003997803, -5.024783611297607, -5.422264575958252, -2.170060634613037, 4.885108947753906, -6.869179725646973, -1.9408087730407715, -0.20594260096549988, -9.470709800720215, -4.134843349456787, 2.6503660678863525, -4.159374713897705, 7.901362895965576, 5.516387939453125, -2.067187786102295, -1.6373590230941772, 1.155415415763855, 1.783582329750061, 1.1805424690246582, 7.397975444793701, -7.5442986488342285, -6.3919997215271, 3.4853739738464355, -3.1635448932647705, 0.4112192988395691, -8.010839462280273, 13.331642150878906, -3.668567419052124, -4.556960582733154, 1.0469684600830078, -5.621485710144043, 5.254175186157227, 0.18645116686820984, 2.7234082221984863, -1.9447689056396484, 6.358335971832275, -7.554687023162842, 0.045891694724559784, 0.03631846606731415, -1.4108389616012573, 0.22078324854373932, -9.138463020324707, 0.06551674753427505, 10.411782264709473, -14.22036075592041, -4.1740827560424805, -8.975263595581055, -2.6087615489959717, 4.394843578338623, -10.887185096740723, -4.913427352905273, 6.235489845275879, 17.718496322631836, -0.11661490052938461, 14.651708602905273, 1.9225850105285645, -8.950471878051758, 8.794088363647461, 0.5462602376937866, 6.3536505699157715, -8.153769493103027, 5.646982669830322, 4.300322532653809, -5.953511714935303, -7.863497257232666, 5.754383087158203, 3.6799721717834473, -2.8882575035095215, -2.183863639831543, 1.4711768627166748, 3.5052154064178467, -4.678380966186523, 6.431110858917236, 4.31274938583374, -1.3850558996200562, -3.5127077102661133, -12.170183181762695, 5.727147102355957, 3.928236961364746, 11.74711799621582, -1.541625738143921, -1.847118616104126, -2.9266817569732666, 0.6445456743240356, 0.9081606864929199, -5.798530578613281, -1.8898813724517822, -12.139152526855469, 2.4627065658569336, 9.039921760559082, 2.5395195484161377, 6.174562454223633, 12.390913009643555, -3.982913017272949, -6.763583183288574, 2.570253610610962, 1.7937850952148438, -6.599144458770752, -1.2737572193145752, 3.072871208190918, -5.576645374298096, 13.469198226928711, -18.800003051757812, 9.23656940460205, -6.173501014709473, 8.97970962524414, -3.3674771785736084, 1.0034133195877075, -2.6493608951568604, 9.042299270629883, -0.2508317232131958, -0.946403980255127, -5.996865749359131, 8.2712984085083, -10.036232948303223, 13.989259719848633, -0.0013234354555606842, -7.8544230461120605, 1.7407571077346802, 9.205704689025879, 1.8351163864135742, -6.213785171508789, 5.439124584197998, 3.990046739578247, -7.904503345489502, 0.7057287096977234, 1.861483097076416, -8.44375991821289, 8.459558486938477, 12.95994758605957, 4.69776725769043, -1.5129190683364868, 8.972602844238281, -0.01308932900428772, -17.71114730834961, 3.5965542793273926, -4.404747486114502, -0.3006945252418518, -4.54971170425415, -4.056035995483398, -3.770744562149048, 13.978760719299316, 0.4669203758239746, -6.273401260375977, 1.427677869796753, -5.290877342224121, 0.06936413049697876, 2.2952663898468018, 5.754252910614014, -2.5364835262298584, 4.008559226989746, -3.231961727142334, 6.930624961853027, 3.114375352859497, 2.932845115661621, 8.953741073608398, -0.44483333826065063, -3.698763608932495, -5.06934928894043, 2.604645252227783, 4.865938186645508, -7.893214702606201, 0.0937640443444252, 5.598154067993164, 0.6417512893676758, 1.0597752332687378, 12.87028694152832, 0.810661256313324, 3.816991090774536, -0.7671364545822144, 1.3219643831253052, -1.9988837242126465, -0.36181360483169556, 4.409139156341553, 2.1026711463928223, 9.788583755493164, -1.5324201583862305, 2.7305145263671875, 6.269411563873291, 2.7559361457824707, -2.8921518325805664, 3.6636736392974854, 3.920677900314331, 3.19445538520813, 0.7736949324607849, 1.7891536951065063, 5.511224746704102, -2.31551194190979, 8.149415016174316, 3.063049554824829, 7.403348922729492, -3.792787551879883, 6.173357963562012, -0.91338711977005, -7.9819231033325195, -4.13888692855835, -3.4782347679138184, 2.6989686489105225, 3.756852388381958, -2.70638370513916, 7.695550441741943, 8.365758895874023, -4.876716613769531, -0.5457528829574585, -0.7223250865936279, -4.983626842498779, -6.7171101570129395, -1.7471213340759277, -0.46561211347579956, 3.673713445663452, 2.1234118938446045, 6.146945476531982, -4.274384021759033, 3.8366854190826416, -14.90523910522461, 2.9595577716827393, -9.046337127685547, 5.505688667297363, -10.00584888458252, 0.5601590871810913, -6.065555095672607, -1.5820726156234741, -8.867020606994629, 5.60034704208374, 5.207279205322266, -2.557687520980835, -7.667445182800293, 14.970003128051758, 2.989023447036743, -1.0410994291305542, 0.8564216494560242, -0.9911004900932312, 2.8445236682891846, 8.118325233459473, -1.7893656492233276, 11.941520690917969, -9.191691398620605, 1.2643719911575317, 1.4771560430526733, -4.37650728225708, 14.0043306350708, 5.202750205993652, 1.2070887088775635, -0.2300630807876587, -9.618643760681152, -6.00324821472168, -2.4054248332977295, -4.976306915283203, -2.829185724258423, 6.011483192443848, -8.671886444091797, 3.3544631004333496, -1.8113220930099487, 0.09383222460746765, 5.96547794342041, 5.167568683624268, 1.681963562965393, -3.140697479248047, 4.895550727844238, 7.4745588302612305, 1.0793648958206177, -7.500591278076172, -12.901365280151367, -3.5006513595581055, 5.449116230010986, 3.873594045639038, 1.005483627319336, 3.8340229988098145, 1.6060113906860352, 2.3574023246765137, 2.6875240802764893, -4.778681755065918, -8.695282936096191, -1.3541021347045898, -12.522696495056152, 8.795488357543945, -3.851790428161621, -5.2837910652160645, 2.751939058303833, 10.351423263549805, 1.4350333213806152, 6.1441168785095215, 0.1431446373462677, -0.21308162808418274, -2.978785753250122, 4.60020112991333, 5.1557793617248535, 7.276215076446533, 4.61760139465332, -3.0233266353607178, -3.718068838119507, 1.8181703090667725, 5.750543594360352, -5.092382431030273, 3.3456180095672607, 3.6235432624816895, 4.547642230987549, -11.555904388427734, 10.654535293579102, -10.942421913146973, -2.037710189819336, 9.33600902557373, 3.95731520652771, 6.090096950531006, 5.555365085601807, -2.400197982788086, 4.942983150482178, -4.193260669708252, -0.15783019363880157, -2.9517126083374023, -10.308018684387207, 3.8057291507720947, -1.5108091831207275, 4.6610894203186035, -17.956100463867188, 11.461221694946289, 4.057580471038818, 4.651520729064941, -1.831111192703247, 8.300470352172852, -8.922607421875, 8.765195846557617, 11.22291374206543, 0.08853829652070999, -0.3717951774597168, -3.9691691398620605, -3.696307420730591, -2.2556331157684326, -6.3624267578125, 0.26883116364479065, -0.6193442344665527, -3.368783950805664, -0.7814370393753052, -4.022617816925049, -2.1704132556915283, 0.5556392669677734, -8.770036697387695, 2.646214723587036, 1.681423306465149, 1.782411813735962, 1.871686339378357, 4.333073139190674, 5.636274337768555, -3.399069309234619, -1.8403639793395996, 2.631074905395508, 4.380691051483154, 0.485141396522522, 1.3963093757629395, -4.631776332855225, -4.588101863861084, 4.044361591339111, 2.1680240631103516, 0.7071276903152466, 0.6364019513130188, 0.8700317144393921, 1.9208133220672607, -5.7531962394714355, -11.107640266418457, 6.150860786437988, 3.2057247161865234, 3.049107551574707, -0.2424613982439041, -3.815114974975586, 10.557084083557129, -2.010718584060669, -10.063531875610352, 2.4326887130737305, -9.527822494506836, -0.24181869626045227, 2.8665361404418945, 4.983570575714111, -1.6477547883987427, -3.3742332458496094, -5.419723987579346, -3.3695435523986816, 1.637481451034546, -3.325993776321411, -2.7408344745635986, -1.5688565969467163, 0.3727010488510132, -12.53402328491211, 2.659306764602661, 1.5036543607711792, 8.995464324951172, -4.611422538757324, 1.5840343236923218, 0.8725786209106445, -6.268546104431152, -6.10651969909668, 9.81358814239502, 13.548868179321289, -6.637622356414795, -6.727476596832275, 2.8742239475250244, 3.6290056705474854, 7.73685884475708, 0.4381158649921417, 3.5784380435943604, -9.932860374450684, -0.00023523345589637756, -3.4266200065612793, 2.611562728881836, -5.920607089996338, 2.251791000366211, -3.1265432834625244, 0.79133141040802, 7.508744239807129, 14.22966194152832, -9.312033653259277, -4.203177452087402, -4.910171985626221, 0.3670448660850525, -5.6189494132995605, 4.9676289558410645, 2.900599956512451, -0.22029876708984375, 1.902640700340271, 10.843489646911621, 15.02386474609375, 6.784565448760986, 6.907181739807129, 4.517200469970703, 1.3442243337631226, 2.575404167175293, 1.6460399627685547, 0.14879977703094482, 8.5471830368042, 2.281561851501465, 0.8349049687385559, 5.990912914276123, -1.4778335094451904, -2.773662805557251, -5.990447044372559, -2.2537002563476562, 4.217107772827148, -0.20592157542705536, 5.261864185333252, 5.562747478485107, 10.570036888122559, 8.7476806640625, 2.646732807159424, -1.8815218210220337, 0.3183755874633789, -3.0815505981445312, -3.263897180557251, 2.6708903312683105, 1.01559579372406, -6.748260498046875, 13.232284545898438, 12.574884414672852, -3.195509433746338, 2.3229610919952393, -8.327624320983887, -0.7458875775337219, 9.577693939208984, -5.560534954071045, 3.0777511596679688, -0.08654629439115524, 0.6014140248298645, 6.904006481170654, 0.488036572933197, 1.5019824504852295, 1.1469979286193848, -1.9278327226638794, 2.3683340549468994, 2.7801616191864014, 3.2070047855377197, -5.732212066650391, -2.069225788116455, 6.051321029663086, 2.865187168121338, 13.885098457336426, 5.780458927154541, -14.521160125732422, 0.11756592988967896, -0.35131821036338806, 2.6294901371002197, 5.263062953948975, -1.414682388305664, -1.6448216438293457, 1.9952610731124878, 0.9757877588272095, -0.6036860346794128, 1.7244853973388672, 2.7298457622528076, 1.1884366273880005, 10.08687686920166, -3.5306966304779053, 7.490076541900635, 0.5075019598007202, -12.563129425048828, 3.658879041671753, -2.144432306289673, 7.2294111251831055, -8.26856803894043, 5.82511043548584, 9.97976016998291, 0.9984720349311829, 1.9993444681167603, 11.25763988494873, 12.694682121276855, 4.003479957580566, 1.4489554166793823, 1.3939292430877686, 5.086729526519775, -8.265958786010742, 1.9186925888061523, 10.98832893371582, 9.937155723571777, 3.106621265411377, 0.9140448570251465, -9.985847473144531, 7.27067756652832, 2.942049503326416, -10.274858474731445, 1.0279624462127686, -5.923027515411377, 5.311679363250732, -10.302014350891113, 9.496743202209473, 0.014906330034136772, 0.7953363060951233, 6.664856433868408, 1.7285492420196533, 7.454432487487793, -8.776168823242188, 7.36884880065918, -5.125316619873047, 2.324564218521118, 4.160977840423584, -7.82302713394165, -7.661965370178223, 0.7825186252593994, 5.980783939361572, 5.053958415985107, -6.391312599182129, 0.2426658421754837, -1.4538383483886719, -4.394611835479736, 8.616450309753418, 0.5061224102973938, 3.839311122894287, 1.0563193559646606, -9.579691886901855, 4.051815986633301, -5.579192161560059, 7.467623233795166, -10.763815879821777, -2.2057065963745117, 12.0824556350708, 0.06719006597995758, -2.386188268661499, 2.8146724700927734, -0.10318484902381897, 6.619422435760498, -2.3569483757019043, -1.4739036560058594, 12.905341148376465, 7.347015857696533, -9.51182746887207, 5.4695353507995605, -6.611915588378906, 1.792946457862854, 2.029416084289551, 1.3879358768463135, -4.031617641448975, -15.098637580871582, -0.1419840157032013, -5.742667198181152, -2.4926538467407227, -5.4162797927856445, -10.061604499816895, -2.435598850250244, 3.875640869140625, 7.2700958251953125, -3.0327413082122803, 0.689306378364563, -4.562466621398926, -4.99839448928833, 4.714637756347656, 0.2860361337661743, 12.917407989501953, 9.088833808898926, -2.3413796424865723, 4.815978527069092, -0.6631961464881897, -2.4884023666381836, -6.856595993041992, -8.46001148223877, 0.8733039498329163, 5.256031036376953, 2.4277405738830566, 15.03919506072998, 1.8474448919296265, -1.7570276260375977, 0.0010030101984739304, -7.252105236053467, -5.16981840133667, 2.3126039505004883, -5.871572017669678, -5.396152019500732, -3.698380947113037, 2.6407675743103027, 9.841073036193848, 0.08929847180843353, -11.339113235473633, -5.2965545654296875, -2.230783462524414, 1.3276052474975586, -11.142233848571777, -7.107847213745117, 2.173227548599243, -1.6446032524108887, -11.547932624816895, 4.666565418243408, 9.655556678771973, -8.234305381774902, -11.615084648132324, -10.727204322814941, -3.489971399307251, 7.461319446563721, 12.12662124633789, -8.693761825561523, -9.4491548538208, -6.030603885650635, 9.723625183105469, 3.694119930267334, 1.6933517456054688, 1.1734724044799805, 5.964296817779541, -6.0610575675964355, 4.979347229003906, -3.0425288677215576, 6.489078521728516, 4.886364459991455, -2.549853801727295, -4.064859390258789, 10.51246166229248, -1.329464077949524, -1.8657788038253784, -8.283913612365723, 1.2448230981826782, 2.806605339050293, 1.1168968677520752, -2.7540130615234375, -0.3055870234966278, 0.4643688201904297, 6.902618408203125, -0.560028612613678, -9.126105308532715, -7.904636383056641, 1.4777259826660156, -5.858426094055176, 0.34194454550743103, -4.677889347076416, -8.028292655944824, -8.241867065429688, -7.755751132965088, -6.980168342590332, -3.5991625785827637, -1.9738765954971313, 6.9705634117126465, -3.368607759475708, 5.068955898284912, -3.619209051132202, 6.603989124298096, -3.8655261993408203, -3.0089468955993652, 13.276073455810547, 0.6370863914489746, 7.625504016876221, 3.8406569957733154, -3.6200006008148193, 8.368823051452637, -1.016006588935852, 1.1496901512145996, 6.657357692718506, 1.410441517829895, -6.501104831695557, 4.80044412612915, -4.3682026863098145, -0.6819925904273987, 6.484899520874023, -8.681910514831543, -1.9289294481277466, -0.13967996835708618, -6.750144004821777, 8.964062690734863, 0.8272517323493958, -9.856642723083496, -0.3768969178199768, -6.835537910461426, 4.464008331298828, -4.939449787139893, -5.72560977935791, 0.21243272721767426, -1.5491952896118164, 3.2272591590881348, 0.2953685522079468, 5.721952438354492, -6.0752129554748535, 8.63851547241211, 6.2397589683532715, -4.67124080657959, 3.856773853302002, 5.908767223358154, -2.9128799438476562, -11.788168907165527, 1.7208380699157715, -8.933032035827637, 2.3441061973571777, 0.5632643103599548, -2.0495030879974365, -4.193301677703857, -13.05734920501709, 0.3077148199081421, -1.4668740034103394, 2.73796010017395, 6.11764669418335, 8.351785659790039, -4.427948474884033, 10.028619766235352, 1.1304832696914673, -5.048563003540039, -3.842257499694824, 3.835322380065918, -5.350214958190918, 2.1239144802093506, -7.95062255859375, -0.8960642218589783, -6.5365424156188965, 2.428790807723999, 0.08847805857658386, -0.965187668800354, 4.839483261108398, -2.1413893699645996], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.8759765625, 1.0732421875, -0.0980224609375, -0.093505859375, -0.7314453125, -0.57958984375, 0.60595703125, -0.26611328125, -0.65478515625, 0.7587890625, -0.338134765625, 0.420166015625, 0.5068359375, 1.0244140625, 0.005626678466796875, -1.078125, 1.0009765625, -0.75244140625, -0.173828125, 2.458984375, 0.8134765625, -0.28515625, -2.041015625, -0.89013671875, 0.849609375, -0.8271484375, -0.41943359375, -1.4677734375, 1.9404296875, 0.295166015625, 2.060546875, -1.6689453125, 1.1474609375, -0.9111328125, 0.9248046875, -1.2255859375, 0.03411865234375, 1.7529296875, 0.537109375, 0.72802734375, 0.509765625, -0.05084228515625, -0.476806640625, -0.88330078125, 0.279541015625, 0.3994140625, -1.34765625, -0.201904296875, -1.5869140625, 0.49609375, -0.06103515625, -0.1644287109375, 0.146240234375, 1.623046875, -0.7109375, -0.30615234375, 0.87158203125, -1.787109375, -0.513671875, 0.35546875, 0.88916015625, -0.41943359375, 1.4853515625, -0.20361328125, -0.282958984375, 2.095703125, 0.346923828125, -1.01953125, 0.214599609375, 1.740234375, -0.728515625, -0.7685546875, 0.385009765625, -0.239501953125, -1.1435546875, 0.94482421875, 0.86767578125, 0.141845703125, -0.51318359375, 0.50439453125, 0.71923828125, -0.66259765625, -0.1103515625, -1.923828125, -1.494140625, 0.04290771484375, -0.446044921875, -2.0078125, -2.4609375, -0.9453125, 0.2484130859375, -1.060546875, 0.9189453125, 0.982421875, -1.490234375, 0.19775390625, 0.043853759765625, -1.07421875, 1.7978515625, 0.806640625, 0.171875, 0.736328125, 0.7685546875, -0.1646728515625, 0.065673828125, 0.66162109375, -1.43359375, -1.2060546875, -0.69287109375, 0.56298828125, 0.61083984375, 0.6884765625, -0.00547027587890625, 1.7822265625, 0.96728515625, -1.140625, 0.5361328125, 0.9638671875, 2.087890625, -1.029296875, -0.3974609375, 0.351806640625, -0.220458984375, -0.40185546875, 0.01019287109375, -0.50048828125, -1.146484375, 0.169677734375, 0.861328125, -0.131103515625, 1.1826171875, 0.9560546875, 0.203857421875, -0.357177734375, -0.83349609375, -0.93701171875, 0.454345703125, -0.1845703125, 0.5263671875, 0.703125, 1.048828125, 0.332275390625, -0.494873046875, 0.0618896484375, -0.64208984375, 0.03167724609375, -0.447998046875, -0.43115234375, 0.024566650390625, 0.489990234375, 1.517578125, 0.634765625, 0.31005859375, -0.61328125, 1.1767578125, -0.398681640625, -0.0159759521484375, 1.0205078125, -1.0107421875, 0.9052734375, -0.454833984375, -0.8076171875, -0.449951171875, -0.49658203125, -1.0126953125, -0.1624755859375, -0.71630859375, -1.5546875, -0.88671875, 0.52001953125, -1.060546875, 0.0183258056640625, 0.916015625, 1.9443359375, -0.231689453125, -0.325927734375, -0.603515625, 0.7099609375, 0.85009765625, 0.93994140625, 2.279296875, -1.2041015625, 0.489013671875, 0.9638671875, -0.00012576580047607422, 0.25390625, -0.430908203125, -0.07867431640625, -0.0421142578125, -0.245361328125, 0.1829833984375, 0.525390625, -0.72509765625, 0.8544921875, -2.03515625, -1.0654296875, -1.0244140625, -0.472900390625, 1.40625, -0.1060791015625, -0.02581787109375, -0.85107421875, 0.5751953125, -0.66845703125, 1.234375, -0.431640625, -0.378662109375, 0.4208984375, 0.307373046875, 1.2412109375, -0.0118560791015625, -0.5, 0.035552978515625, 0.07928466796875, 0.94189453125, -0.190673828125, 0.31640625, 0.49072265625, -0.82080078125, 0.88134765625, -0.281494140625, 1.2626953125, 1.2392578125, -0.278076171875, -0.386474609375, 1.333984375, 0.92578125, 0.1834716796875, 0.1756591796875, -1.3828125, -0.03594970703125, -0.28173828125, 1.744140625, -1.0859375, -1.20703125, -0.6982421875, -2.224609375, -1.1591796875, 0.04888916015625, -1.2783203125, 0.6728515625, 1.818359375, -0.391845703125, -0.12054443359375, 0.55126953125, -0.2447509765625, 0.357421875, 0.46630859375, 0.31396484375, 0.88037109375, -1.017578125, -0.1717529296875, 2.087890625, 1.1279296875, -0.72607421875, 0.041290283203125, -0.0294342041015625, -0.197265625, 1.2412109375, -0.76220703125, 2.18359375, 0.982421875, 0.060455322265625, 1.595703125, 0.9833984375, 1.296875, 1.4765625, -0.239990234375, 1.6806640625, 0.70458984375, -0.0447998046875, 0.8681640625, -0.79833984375, 0.55029296875, 0.81103515625, 0.6181640625, 0.284912109375, -1.8310546875, 0.0174713134765625, -0.169677734375, -0.8955078125, -2.201171875, -1.21875, 0.383056640625, -2.205078125, -0.341796875, 0.11279296875, 1.5107421875, 0.07379150390625, 0.814453125, 1.46484375, 1.55078125, -0.332275390625, 0.9775390625, 1.0654296875, -0.33154296875, 1.2900390625, 0.60009765625, 0.463623046875, 0.317138671875, 0.1973876953125, 0.52734375, 1.5029296875, -1.1884765625, -2.3671875, 0.004993438720703125, -1.7060546875, 0.09759521484375, 0.94189453125, 0.12255859375, -0.115234375, -0.241455078125, 0.91552734375, 0.439208984375, 0.098876953125, 0.340576171875, 0.9423828125, 0.190185546875, 0.794921875, 0.3623046875, -1.0185546875, -1.044921875, -1.435546875, -2.029296875, 1.337890625, -1.236328125, -0.7802734375, -0.85302734375, -0.42138671875, 0.357666015625, 1.05078125, 1.544921875, 2.462890625, -0.26416015625, 1.0859375, 0.201416015625, 0.80517578125, -0.99560546875, 0.42626953125, -2.291015625, -0.330810546875, -0.08880615234375, 0.36572265625, -1.1875, 0.049774169921875, 1.6318359375, -0.400146484375, -3.1328125, 0.98583984375, -1.6123046875, 0.921875, 2.810546875, -0.2139892578125, 1.4189453125, 0.330810546875, 0.79833984375, -1.083984375, -1.1396484375, 0.1529541015625, -1.2265625, 0.474853515625, -2.19140625, 1.3662109375, -1.501953125, 0.31298828125, -0.308349609375, -1.2919921875, 0.43505859375, -0.68701171875, -0.53076171875, 2.060546875, -0.44140625, -0.53173828125, 0.484130859375, 0.176513671875, 0.3759765625, 0.028900146484375, 0.5498046875, -0.290771484375, -0.1737060546875, -0.8154296875, 0.2442626953125, 0.05804443359375, 0.54296875, -0.9423828125, 1.375, -0.35498046875, 0.4931640625, 0.424072265625, -1.529296875, 1.26171875, 0.010162353515625, 1.279296875, 0.308837890625, 1.287109375, -1.306640625, -0.1824951171875, -0.66259765625, -0.63818359375, -0.467041015625, -1.0751953125, 0.433349609375, -0.437255859375, -2.00390625, 0.406982421875, 0.1080322265625, 2.193359375, 0.289306640625, -1.181640625, -1.349609375, 0.81103515625, -0.319091796875, -1.693359375, -0.66064453125, 0.10015869140625, -0.64453125, -0.568359375, -0.49169921875, -1.390625, -0.270751953125, 0.828125, -2.26953125, -0.76806640625, -0.8701171875, -0.4208984375, -1.3701171875, 0.2406005859375, 1.080078125, -0.6845703125, 1.2080078125, -0.4619140625, -1.4560546875, -0.7080078125, 1.044921875, -0.028778076171875, -0.07757568359375, -0.8076171875, 0.468017578125, -0.20947265625, -0.5517578125, -0.215087890625, 1.38671875, 0.3603515625, 0.358642578125, 0.7333984375, -1.287109375, -1.314453125, -1.2900390625, -0.66748046875, 0.060272216796875, -0.63232421875, 1.7490234375, 3.390625, -1.4326171875, -0.09246826171875, -1.0693359375, -0.02081298828125, 0.2919921875, -0.61376953125, 0.35205078125, -0.0260162353515625, -0.66357421875, -1.4580078125, -1.0068359375, -1.0107421875, 0.08868408203125, 1.541015625, 1.7333984375, -0.66796875, 0.08270263671875, 2.54296875, -1.150390625, 0.43310546875, -1.0087890625, 0.642578125, -0.77392578125, -0.367431640625, 1.1123046875, -0.183837890625, -1.2744140625, 1.2734375, 0.60107421875, 1.2119140625, -1.6103515625, 0.3369140625, 1.7451171875, 0.615234375, -0.35009765625, 1.1787109375, 0.284912109375, -1.78125, -1.005859375, 0.0728759765625, -0.77001953125, -0.420654296875, 1.0302734375, 0.80615234375, -1.3046875, -0.2169189453125, -0.27880859375, 0.65087890625, 0.43017578125, 0.064697265625, 0.2393798828125, -1.1357421875, -1.1279296875, 0.66845703125, -0.71240234375, 1.0849609375, 1.2890625, 2.08984375, -0.331298828125, 0.3046875, -1.296875, 1.330078125, -1.05078125, 1.0986328125, 0.0460205078125, -0.343994140625, -1.9833984375, 1.8017578125, -0.708984375, 0.322021484375, 0.322509765625, 0.1815185546875, -1.150390625, 0.96826171875, -0.33935546875, 0.57421875, 2.0390625, 0.2408447265625, -2.40625, -1.44140625, -0.1212158203125, 1.53125, 0.8671875, 2.087890625, 2.84375, -1.0615234375, 0.327880859375, 0.7099609375, 0.420166015625, -0.4091796875, -1.20703125, -0.2467041015625, 0.281494140625, -0.378173828125, 0.64453125, -0.6455078125, -1.8017578125, 1.43359375, 0.52978515625, -1.55078125, 2.162109375, 0.765625, -0.224365234375, -0.8515625, -1.6103515625, 0.047393798828125, 0.6884765625, -0.39599609375, -0.09356689453125, 0.8173828125, 0.35302734375, -0.9384765625, 1.6318359375, -1.36328125, -0.429931640625, 0.74169921875, -0.81982421875, 0.355224609375, -1.421875, 0.86572265625, 0.783203125, 1.01953125, -1.6298828125, 0.9150390625, -1.1533203125, -0.188720703125, 0.344482421875, -0.140625, 0.2308349609375, -0.197509765625, 1.7646484375, 1.0263671875, -1.6044921875, 0.75537109375, -0.44482421875, -0.104736328125, -1.0107421875, -0.40771484375, 2.119140625, 0.1826171875, 1.1982421875, 1.3701171875, 2.28125, 0.42041015625, 0.94140625, 0.6455078125, -0.6572265625, -0.004283905029296875, -0.5732421875, 1.9384765625, -1.2490234375, 0.005588531494140625, 0.21728515625, 0.7119140625, -1.0517578125, -1.232421875, -0.006824493408203125, 0.48046875, 0.58154296875, -1.3017578125, -0.9287109375, -0.1588134765625, -0.09075927734375, -0.06890869140625, -1.078125, 1.3525390625, 0.31396484375, -0.1517333984375, 0.66943359375, 0.106201171875, -0.29541015625, -1.1279296875, -0.95556640625, -3.03125, -0.6611328125, 0.814453125, 0.03546142578125, 0.82373046875, 0.38427734375, 2.37890625, 1.8544921875, -1.171875, 0.6064453125, -0.369140625, 0.77978515625, -0.233154296875, -1.5048828125, 1.982421875, 0.2242431640625, 0.478515625, -0.51953125, -0.359375, -0.97998046875, 0.92919921875, -0.55126953125, -0.372802734375, -0.382568359375, 2.5546875, -0.0814208984375, 1.0869140625, -0.0880126953125, 1.0966796875, -0.34326171875, -0.1693115234375, 0.65966796875, 0.6259765625, 1.3525390625, 1.1572265625, 0.364990234375, 0.30029296875, 1.677734375, 1.9033203125, -1.865234375, 2.205078125, 0.1575927734375, -1.306640625, 0.89013671875, 0.0916748046875, 0.491455078125, 0.1253662109375, 1.27734375, -0.79150390625, 0.495849609375, -0.0206756591796875, -1.49609375, -0.1063232421875, -0.78173828125, -0.261474609375, -0.1900634765625, -0.67822265625, 0.7509765625, 0.7099609375, 0.389404296875, -0.353271484375, -1.72265625, 1.8095703125, -0.8623046875, -0.7744140625, 0.2161865234375, -0.32666015625, 0.72802734375, 0.401123046875, -1.3544921875, 0.154296875, 3.177734375, -1.8564453125, -0.423095703125, -0.434814453125, 0.7744140625, 2.037109375, 0.2333984375, 0.347412109375, 0.86962890625, 1.1337890625, -0.52880859375, 1.173828125, -0.04302978515625, 1.91015625, -0.69970703125, -0.75341796875, 0.1441650390625, -0.00353240966796875, -0.16015625, -0.418701171875, -0.18603515625, -0.28759765625, 1.349609375, 1.4521484375, 1.626953125, 1.32421875, -0.77099609375, -0.884765625, 1.1220703125, -0.41455078125, -1.388671875, 0.002025604248046875, 0.4794921875, -0.447998046875, -1.7001953125, 1.1259765625, -0.002758026123046875, -0.67236328125, 0.65673828125, 0.346923828125, 1.7529296875, -2.173828125, 0.95068359375, -0.404541015625, -1.1279296875, -0.58984375, 0.3408203125, -0.341064453125, -0.78759765625, 0.685546875, 0.58984375, 1.0146484375, 1.4052734375, -0.705078125, -0.434814453125, 1.80078125, -1.5478515625, -1.0224609375, -0.7138671875, -0.5400390625, -0.466552734375, 0.61328125, 0.5087890625, -2.11328125, -1.2294921875, 0.94384765625, 0.0699462890625, -0.73193359375, -0.308349609375, 0.305908203125, 0.623046875, -1.7939453125, -0.10107421875, -1.4521484375, 0.41796875, -2.388671875, -0.55859375, 0.2294921875, 0.65625, -1.26953125, -0.237060546875, -0.78271484375, 1.4638671875, 1.7060546875, -1.13671875, 1.58984375, 1.388671875, -0.329833984375, -0.7626953125, 0.267822265625, -0.299072265625, -0.01546478271484375, 0.173583984375, 0.23193359375, 0.478515625, -1.146484375, -1.515625, -0.2113037109375, -0.414794921875, -0.91748046875, 0.2098388671875, 1.423828125, 0.7353515625, -2.13671875, -0.4150390625, -0.4697265625, 0.96826171875, -0.18798828125, 0.27490234375, -0.4326171875, -1.0224609375, -0.78515625, 0.040313720703125, 0.9208984375, -0.6025390625, -0.037750244140625, 0.006072998046875, -0.89453125, 0.165771484375, 0.2130126953125, -0.09222412109375, -0.75048828125, 1.3203125, -1.1142578125, 0.2308349609375, -0.8076171875, -0.96240234375, -1.5859375, -0.304443359375, 0.286865234375, 1.09375, 1.1025390625, -0.2919921875, -1.626953125, -0.1727294921875, 0.78564453125, 0.00737762451171875, -1.068359375, 0.1859130859375, -0.35546875, 0.46826171875, -0.291748046875, -0.271728515625, -0.325439453125, -1.8046875, 0.71875, -0.740234375, 2.291015625, -1.201171875, -0.15380859375, -0.297119140625, -1.921875, -0.90625, 0.1724853515625, 1.6474609375, 0.5576171875, -1.794921875, -1.337890625, 0.216796875, 1.2333984375, -0.3203125, 0.7490234375, 1.9931640625, 0.71826171875, -0.62646484375, 0.7021484375, -0.79052734375, -0.2176513671875, -1.0927734375, -0.00324249267578125, -0.97509765625, -0.80126953125, -0.70947265625, -1.0654296875, -1.1064453125, 1.64453125, 0.54052734375, -0.78515625, 1.193359375, -1.1650390625, 1.251953125, 0.2158203125, -0.3759765625, 0.8115234375, -0.75830078125, -1.3837890625, 2.1640625, -0.623046875, -1.724609375, -1.353515625, 0.7705078125, -2.4296875, -0.3349609375, 1.5537109375, 0.8662109375, -0.298095703125, -0.86083984375, -2.267578125, 0.99951171875, 1.21875, -0.34423828125, 0.69970703125, 0.69189453125, -0.89501953125, -0.2203369140625, 0.266845703125, 2.505859375, 0.39404296875, 1.259765625, -0.359619140625, 1.81640625, -1.65625, 1.232421875, 0.5849609375, -0.37841796875, -0.56591796875, -0.343994140625, -1.0556640625, -0.603515625, 0.916015625, 0.64306640625, 1.0439453125, -0.56201171875, 0.896484375, 0.56005859375, -0.58984375, 0.1412353515625, 0.05517578125, 2.166015625, -0.79443359375, 1.3935546875, 0.044891357421875, -0.54296875, 0.1907958984375, -0.44677734375, -0.57861328125, -2.662109375, -1.77734375, 0.058563232421875, -1.6611328125, -1.3076171875, -0.0219573974609375, 0.2568359375, -0.80517578125, 0.22509765625, 0.06256103515625, 1.27734375, 0.7568359375, 0.36962890625, 0.59716796875, 2.205078125, 1.8447265625, -1.2255859375, -0.42041015625, 0.4072265625, -2.216796875, 0.19580078125, -0.505859375, -0.05194091796875, -0.98681640625, -0.211181640625, 0.1644287109375, -0.55810546875, 0.0484619140625, 0.892578125, -0.61767578125, -0.0156097412109375, -0.1795654296875, -0.121337890625, -0.2249755859375, -0.309814453125, -0.194580078125, -0.60595703125, -3.083984375, 1.919921875, 0.51611328125, -1.37109375, 0.315185546875, -0.388671875, 0.53271484375, -0.1876220703125, 1.0390625, 0.57421875, -0.32763671875, 0.0819091796875, -0.68359375, 0.0297393798828125, 1.4560546875, 0.1832275390625, -0.703125, 1.2021484375, 0.5712890625, 0.014404296875, 0.70947265625, -0.912109375, -0.77001953125, 0.77587890625, -0.95849609375, 0.9296875, -1.1025390625, -1.224609375, -0.77490234375, 2.7109375, 1.052734375, -1.09375, -0.57080078125, -0.163330078125, 0.74755859375, 0.52197265625, -1.8544921875, -0.66064453125, -0.421630859375, -0.37158203125, -0.174072265625, 0.02392578125, -0.65185546875, 1.29296875, 0.54052734375, 0.9638671875, 1.01953125, 0.053924560546875, -0.0731201171875, -0.69091796875, -0.292236328125, -0.0003218650817871094, -0.7451171875, -0.43798828125, 0.38232421875, 0.323974609375, -1.0634765625, -0.072998046875, -1.4306640625, 0.0160064697265625, 0.81396484375, -0.7333984375, -0.003910064697265625, -0.83154296875, -0.73046875, -0.215576171875, 1.5341796875, 0.29296875, -0.382568359375, 1.2333984375, -0.580078125, 1.3603515625, 0.63916015625, -0.7666015625, -0.19775390625, -0.294921875, 1.6669921875, 0.343017578125, 0.75732421875, 1.2060546875, 0.314697265625, -0.82177734375, 1.5615234375, 2.177734375, 1.109375, 0.09124755859375, -0.2744140625, 0.1402587890625, 0.7314453125, -1.357421875, 1.083984375, -0.67626953125, 0.437744140625, 1.583984375, 0.44873046875, -0.36962890625, 0.66650390625, -1.1123046875, 1.0712890625, -0.2408447265625, 0.203369140625, 0.87841796875, -0.9951171875, 0.35546875, 0.043701171875, 0.159423828125, 0.5595703125, 0.0177459716796875, 1.396484375, 1.1240234375, -1.193359375, -0.56884765625, -0.873046875, -0.6904296875, 0.09490966796875, 0.27294921875, 0.669921875, 0.52880859375, 0.81298828125, -0.37158203125, 0.1993408203125, -1.46484375, -0.31689453125, 0.59228515625, -2.099609375, 0.2802734375, -0.85595703125, 1.39453125, 0.294189453125, 0.9140625, -1.5595703125, -1.1513671875, 1.0810546875, -1.2373046875, -1.9521484375, -1.1083984375, -0.8212890625, -0.1134033203125, -1.498046875, 0.251953125, -1.0341796875, -1.259765625, 1.2001953125, -0.3935546875, 0.6630859375, 1.044921875, 1.47265625, -0.1845703125, -0.207275390625, -1.31640625, 1.060546875, -0.7119140625, -1.931640625, -0.05419921875, -1.4755859375, 0.1136474609375, 0.0390625, 1.0205078125, 0.410888671875, -1.0185546875, -1.4580078125, -0.53076171875, -0.02606201171875, -0.59326171875, -0.173583984375, -1.5927734375, -0.4404296875, -0.54150390625, 0.11602783203125, -1.884765625, 2.236328125, -0.27783203125, -1.3447265625, -1.1083984375, 0.2342529296875, 0.052337646484375, -0.67822265625, 0.9814453125, -0.37744140625, 0.7197265625, -0.300537109375, -1.8154296875, 0.40380859375, 1.8662109375, -0.5869140625, 0.712890625, 0.5537109375, 1.2783203125, 1.443359375, -0.56982421875, -1.4814453125, 0.25830078125, 0.00991058349609375, 0.2113037109375, 0.99560546875, 0.374755859375, 0.430908203125, 0.728515625, 0.15966796875, 0.031768798828125, 2.19921875, -0.04058837890625, 0.0755615234375, -1.560546875, 0.1961669921875, 0.6455078125, 0.331298828125, 1.248046875, -1.3310546875, 0.5458984375, 0.87353515625, 0.2281494140625, 0.1761474609375, 1.30859375, -0.04547119140625, -1.111328125, 0.548828125, -1.2490234375, 1.54296875, -1.365234375, 1.609375, 0.90478515625, 1.2275390625, 0.66015625, -1.5654296875, 0.1207275390625, 1.3798828125, 1.744140625, 0.262451171875, 1.6279296875, -1.3193359375, 1.3369140625, -0.67236328125, -0.1146240234375, -0.320068359375, -0.208740234375, -0.011993408203125, -1.630859375, -1.8984375, -0.96728515625, 0.10894775390625, -2.095703125, 0.8701171875, -0.48046875, -0.007259368896484375, -0.0180511474609375, 1.1357421875, 1.154296875, -0.58837890625, -0.505859375, 0.4208984375, -0.7080078125, 1.3310546875, 0.0797119140625, 0.0224151611328125, -0.224365234375, 0.787109375, 0.63037109375, 0.63427734375, 2.0, -1.3251953125, -1.17578125, -0.320556640625, 0.70849609375, 0.5166015625, -0.41845703125, -1.4072265625, -0.0169677734375, -0.1273193359375, -1.5107421875, 0.370849609375, -0.5263671875, -1.484375, 1.4580078125, 0.397216796875, 0.09326171875, 0.2998046875, 0.66064453125, 1.1484375, -0.5703125, -0.8525390625, -0.79345703125, -0.5888671875, -0.970703125, -0.703125, -0.2435302734375, -0.62109375, -0.3671875, 1.1591796875, -0.53466796875, -1.6005859375, -1.6435546875, -0.289306640625, 0.513671875, -0.30810546875, 0.89501953125, 0.402587890625, -0.106689453125, -0.37109375, -0.37939453125, 0.5048828125, 1.5810546875, -1.4921875, 0.05535888671875, -0.16748046875, -0.318115234375, -0.129638671875, 0.45751953125, -1.056640625, 1.5595703125, 0.52294921875, 0.6513671875, -0.703125, 0.0243377685546875, -1.1181640625, 0.5009765625, 1.1083984375, -2.259765625, -0.9130859375, 0.52880859375, -1.58203125, 1.54296875, 2.083984375, -0.6689453125, -1.4267578125, 0.70556640625, -0.00647735595703125, 0.72314453125, -2.33203125, -1.0419921875, -0.185302734375, 0.01270294189453125, 0.1343994140625, 1.09765625, -0.0802001953125, 0.6787109375, -1.5322265625, 1.3896484375, 0.70849609375, 1.369140625, -0.032135009765625, 0.7763671875, -0.4462890625, -0.1566162109375, 0.7431640625, -0.50146484375, -1.1162109375, 0.024383544921875, 0.152587890625, -0.673828125, 1.1357421875, 0.67529296875, -0.287353515625, -1.9775390625, 1.33203125, -0.67431640625, 1.4716796875, -0.289794921875, -0.8515625, 0.517578125, 0.20751953125, -0.0003159046173095703, -0.473876953125, 1.5849609375, 0.401611328125, 0.09222412109375, -2.1015625, 0.9912109375, 0.44189453125, -0.78125, 0.82373046875, 0.50927734375, -2.244140625, -0.4794921875, 0.349853515625, -0.131591796875, -0.06097412109375, -0.62744140625, -1.359375, -1.56640625, -1.9150390625, 0.70263671875, -0.5107421875, 0.75439453125, -0.482666015625, 1.841796875, -0.275634765625, 0.390869140625, -0.91259765625, -0.007793426513671875, -0.058197021484375, -0.53857421875, 1.9423828125, -1.4619140625, -0.1556396484375, 0.2335205078125, 0.193359375, 1.3896484375, -0.3916015625, 1.6162109375, -0.3447265625, -0.026763916015625, 1.603515625, -0.46435546875, -0.27685546875, -0.58740234375, -0.4208984375, -0.439208984375, -0.0928955078125, 2.80859375, -0.2115478515625, -0.435302734375, -0.2430419921875, -1.8134765625, 0.5263671875, -1.189453125, -2.41015625, 0.75048828125, 0.36572265625, -2.037109375, -0.2880859375, -0.58056640625, -0.45703125, -0.35107421875, -2.439453125, -0.17431640625, -0.61376953125, 0.7705078125, -0.7626953125, 0.265625, 0.04071044921875, -0.595703125, 1.2392578125, -1.291015625, -0.48779296875, -0.7177734375, -0.087646484375, -0.18212890625, -2.15234375, 1.5849609375, -0.59912109375, 0.7470703125, 1.0673828125, 0.57958984375, -1.677734375, -0.1893310546875, -0.04766845703125, 0.2108154296875, -0.80712890625, -0.360595703125, -0.6728515625, -1.4140625, -0.513671875, -0.353759765625, -1.1513671875, -0.159423828125, 0.450439453125, 0.282958984375, 2.515625, 0.10205078125, 1.888671875, 0.080810546875, -0.401611328125, -1.4599609375, -0.330322265625, -1.1220703125, 0.83544921875, -1.75, -0.46923828125, -1.5390625, 2.146484375, -2.138671875, 0.023223876953125, -0.32470703125, -0.7060546875, -0.90380859375, 0.434814453125, -0.2098388671875, 0.02301025390625, 0.43701171875, 0.31103515625, 1.4052734375, 0.56884765625, 0.1492919921875, 0.367431640625, 0.360595703125, -0.56884765625, 0.1971435546875, -0.025604248046875, -0.9375, -1.08203125, 0.268798828125, 0.97998046875, -0.167724609375, 0.06494140625, -0.419677734375, -0.3134765625, -1.3076171875, -0.4248046875, 0.345458984375, 0.1600341796875, -0.90869140625, -0.463623046875, 0.62158203125, -0.8310546875, 0.83056640625, 0.1676025390625, 1.5224609375, -0.1065673828125, -0.005214691162109375, -0.26708984375, -0.28759765625, 0.6484375, 0.224609375, -0.81396484375, 0.353759765625, -2.330078125, 0.58203125, 0.92236328125, 1.3330078125, 0.450927734375, 0.0030269622802734375, -0.78173828125, 0.0496826171875, 0.97412109375, -0.26220703125, 0.64892578125, -0.748046875, 0.1103515625, -1.23046875, 0.69482421875, -1.947265625, -0.00616455078125, -0.04400634765625, 0.378662109375, 1.94921875, 0.93701171875, -1.892578125, -1.24609375, 0.67724609375, -2.080078125, -0.1610107421875, 0.56396484375, 1.0146484375, 0.163818359375, -0.12548828125, 1.5361328125, 1.72265625, 1.3857421875, 1.162109375, -0.1627197265625, -0.8466796875, 1.412109375, 0.97314453125, -0.3447265625, 1.1201171875, 0.274658203125, -0.046356201171875, 0.57177734375, 0.951171875, -1.09765625, 1.052734375, 0.1927490234375, 0.7236328125, -0.86865234375, 1.4873046875, -0.050994873046875, 0.2291259765625, -1.126953125, -0.306396484375, -0.900390625, -0.32421875, -2.791015625, 1.2900390625, 0.41162109375, -0.9013671875, -1.384765625, -0.82080078125, -1.1416015625, -0.2193603515625, -0.089111328125, 0.0819091796875, -0.218505859375, -0.323974609375, 0.447021484375, 0.73046875, -0.5087890625, 0.8330078125, -0.958984375, -0.276611328125, 1.03125, 0.90576171875, -0.38623046875, -1.2490234375, -0.95361328125, -0.491943359375, -1.0615234375, -1.697265625, 2.005859375, -0.50732421875, -0.489990234375, 0.25244140625, 2.01171875, -1.171875, -0.96044921875, -0.31640625, 2.109375, -0.51611328125, 0.61767578125, -1.6240234375, -0.337646484375, -0.03692626953125, 0.08734130859375, -0.408935546875, 0.00704193115234375, 0.28173828125, 0.5078125, 0.1519775390625, -0.52001953125, -1.783203125, -1.25, -1.5302734375, -0.84375, -1.4189453125, 0.240966796875, -0.34765625, -0.2227783203125, 0.359130859375, -0.876953125, -0.5146484375, -1.6591796875, 1.0751953125, -0.3916015625, -0.96240234375, -0.38037109375, 1.3232421875, -0.329833984375, -0.88427734375, -1.71875, 0.8681640625, 0.7314453125, 0.364013671875, 0.30078125, 0.7021484375, 0.98486328125, -1.748046875, 1.19921875, 0.84619140625, -0.560546875, -0.78466796875, -2.505859375, 0.525390625, 1.375, -0.71337890625, -1.181640625, 0.7763671875, -0.84228515625, 0.1236572265625, 0.9150390625, -1.4873046875, 0.098876953125, -0.49560546875, -0.97412109375, -0.8359375, -0.0029277801513671875, -1.5283203125, 0.89013671875, -0.2763671875, 0.63525390625, 0.2032470703125, -0.5576171875, -1.8349609375, 1.6240234375, 0.483154296875, -1.662109375, -0.55810546875, 0.56103515625, 1.4765625, -1.6044921875, -0.08978271484375, 0.1636962890625, -2.6015625, -1.9208984375, -1.63671875, 1.626953125, -0.251953125, 0.60400390625, -1.146484375, -0.013214111328125, 0.55615234375, 0.57177734375, -2.04296875, 3.044921875, 0.32080078125, -0.98876953125, -0.042236328125, 0.10186767578125, 0.404296875, -2.130859375, 0.626953125, -0.19873046875, 0.100341796875, -2.298828125, 0.87841796875, 1.5556640625, 0.6533203125, -0.09149169921875, -1.8369140625, 0.75439453125, -1.2119140625, 0.9736328125, 0.8505859375, -1.3935546875, 0.07244873046875, -0.340576171875, -0.55615234375, -0.90185546875, 1.3701171875, 1.2705078125, 1.0185546875, -0.2568359375, -0.9140625, -0.0003535747528076172, -0.9423828125, -0.0653076171875, 1.1767578125, 0.36572265625, 1.87109375, -0.276123046875, 1.1162109375, 0.4892578125, -0.2457275390625, 0.33447265625, 0.0273590087890625, -1.162109375, -0.10601806640625, 0.059356689453125, -0.90966796875, -0.2396240234375, -0.8603515625, -0.2197265625, -1.123046875, 0.031341552734375, -1.259765625, -0.298583984375, -1.8828125, 1.138671875, 1.677734375, 1.4033203125, -0.72314453125, 0.3408203125, 1.8623046875, -0.349609375, -1.47265625, 0.66650390625, -0.72412109375, -0.39794921875, -0.283203125, -0.59375, 0.1729736328125, -0.2109375, 0.8603515625, 0.9638671875, -2.357421875, -1.0556640625, -0.57373046875, -0.468017578125, -1.4990234375, -0.32080078125, 1.44140625, 0.016571044921875, -0.2103271484375, -0.44384765625, -0.143798828125, -0.0469970703125, -0.36669921875, 0.17431640625, 1.107421875, 0.93896484375, 0.043304443359375, 0.484130859375, 0.67431640625, -0.69384765625, -0.55859375, 0.198486328125, -3.001953125, 1.892578125, -0.1380615234375, 2.033203125, -0.564453125, 0.28759765625, -1.02734375, 0.06341552734375, -0.06781005859375, 1.8515625, 0.45849609375, 1.1171875, -0.202392578125, 1.2548828125, -0.6376953125, 0.448486328125, -1.279296875, -0.54638671875, 1.1865234375, -0.8466796875, -1.267578125, 0.40380859375, 0.42333984375, 0.9169921875, 0.041534423828125, 3.419921875, 1.140625, 1.5087890625, -0.861328125, 1.232421875, -0.5087890625, 1.626953125, 1.677734375, 0.95458984375, 1.041015625, -0.74755859375, -0.2073974609375, 2.126953125, -0.283447265625, -0.5703125, 1.607421875, 1.07421875, -0.7392578125, 0.252197265625, -0.458251953125, 1.638671875, -0.52685546875, 0.06658935546875, -0.1259765625, 0.1490478515625, -1.8525390625, 0.41357421875, 1.6708984375, 0.078369140625, -0.328857421875, 0.09979248046875, 1.416015625, 1.6455078125, -0.6904296875, 0.7294921875, -0.65185546875, -0.10833740234375, 0.03240966796875, -0.1497802734375, -0.2354736328125, -1.40234375, -0.326171875, 0.58056640625, 1.060546875, -0.33203125, 1.060546875, -0.2666015625, -0.4501953125, 0.25, 1.9736328125, -0.8623046875, -0.83935546875, -0.93603515625, 0.71337890625, -0.391845703125, -0.1158447265625, -0.9013671875, 1.130859375, -0.1494140625, -0.982421875, -0.65283203125, 0.1649169921875, -0.0462646484375, 0.55126953125, 0.34521484375, -0.201904296875, 2.380859375, 0.039337158203125, -0.02252197265625, 0.56591796875, -0.55615234375, -1.1328125, 0.77734375, -0.90673828125, 0.0050811767578125, 0.3818359375, 0.34375, -0.189208984375, 0.54638671875, -0.00959014892578125, 0.984375, 1.576171875, -0.24560546875, 1.0283203125, -1.4521484375, 1.4404296875, 0.041839599609375, -0.02459716796875, -1.0390625, -0.86962890625, -0.857421875, 0.8525390625, -1.77734375, 0.8994140625, -0.1209716796875, 1.744140625, 0.8837890625, 0.2130126953125, -0.2236328125, 1.169921875, 0.51708984375, 0.1751708984375, 0.888671875, 0.355224609375, -0.0982666015625, 0.8251953125, -0.21630859375, 0.034942626953125, 0.29150390625, -0.454345703125, 0.751953125, 0.37890625, -1.984375, -0.5009765625, -0.1439208984375, 0.1524658203125, -0.064697265625, 1.4560546875, 0.68701171875, -0.253173828125, -0.51318359375, 0.73583984375, -1.607421875, 1.1162109375, -0.375, 1.0205078125, -0.31787109375, 0.2159423828125, -0.240966796875, -0.485595703125, 0.515625, -0.646484375, 1.32421875, -1.712890625, 0.72900390625, -1.14453125, -0.444091796875, -1.9384765625, 0.08856201171875, -1.2392578125, 0.9384765625, 0.045806884765625, -2.412109375, -0.0836181640625, -0.133544921875, -1.2373046875, -0.456298828125, -0.141845703125, 0.345947265625, -0.191650390625, 0.2039794921875, 0.85107421875, 0.018951416015625, 0.96044921875, 0.246337890625, 0.0919189453125, 0.9150390625, 0.93359375, 1.265625, 0.39990234375, -0.329833984375, -0.63330078125, -0.9111328125, -0.2362060546875, 0.97900390625, -1.2978515625, -0.403076171875, -1.3681640625, 0.57421875, 1.0625, -0.1953125, 0.0302581787109375, 0.19189453125, 0.0521240234375, 0.8916015625, 1.8369140625, -0.935546875, -0.9755859375, -0.450439453125, 0.69873046875, -0.262939453125, 0.52783203125, 1.37890625, -0.70361328125, -0.79931640625, -1.787109375, -1.818359375, 0.46728515625, -0.1995849609375, 1.41015625, 0.64599609375, -0.12017822265625, -2.25390625, 0.56982421875, -2.357421875, 0.85205078125, 1.296875, -0.337890625, -2.001953125, 0.77685546875, -0.70263671875, 0.317626953125, -2.91796875, -0.93798828125, -2.404296875, 0.392578125, -1.0263671875, -0.513671875, -0.11676025390625, -0.6025390625, 0.78173828125, 0.66796875, 0.88330078125, -1.9951171875, 0.0063934326171875, 0.1597900390625, 1.03515625, 0.043670654296875, 2.21875, 0.2427978515625, -0.56494140625, 1.27734375, -1.0859375, -1.140625, -0.472412109375, 1.4541015625, -0.8974609375, -1.3828125, 1.546875, 0.79541015625, 1.0283203125, 0.6015625, 1.01171875, 0.68701171875, -0.11859130859375, -0.79443359375, 1.0166015625, -0.677734375, 0.8935546875, 1.876953125, -1.23046875, 1.056640625, 1.0361328125, 0.5732421875, 0.449951171875, 0.974609375, 0.333251953125, -0.4560546875, 0.24755859375, -2.099609375, 1.396484375, 0.09210205078125, 1.1572265625, 0.11102294921875, 0.6455078125, -0.7353515625, 0.341064453125, 1.6728515625, 0.38037109375, -1.796875, 0.919921875, -1.4599609375, -1.109375, -0.52880859375, 1.572265625, -0.7939453125, -0.6328125, 1.078125, 0.5166015625, -1.0703125, -0.69775390625, 1.4921875, -1.1435546875, 1.8935546875, 1.9873046875, -0.759765625, -0.34521484375, -0.179931640625, 0.0693359375, 0.202392578125, -0.61962890625, -0.1026611328125, -0.396728515625, 0.1884765625, -0.099609375, -1.4130859375, -1.12890625, 0.150634765625, 0.1834716796875, 0.890625, 0.09234619140625, 0.13818359375, 0.0352783203125, 2.0234375, 0.7783203125, -0.2861328125, -0.72119140625, 0.0077667236328125, 0.20849609375, 0.149169921875, 1.90234375, 0.2440185546875, 0.5947265625, 1.08203125, -0.334228515625, 1.169921875, 0.63720703125, 0.006313323974609375, -0.398193359375, -0.47607421875, -1.2666015625, -1.6005859375, -0.83251953125, 1.76953125, -1.4638671875, -0.8828125, -0.072265625, 1.04296875, 0.1978759765625, -1.111328125, 2.08203125, -0.059844970703125, 1.4326171875, -1.3369140625, 0.666015625, -0.336181640625, -0.52587890625, -0.97314453125, 0.8798828125, -1.0859375, 0.2939453125, -1.1806640625, -0.26904296875, -1.388671875, 0.5166015625, 1.5810546875, -0.76611328125, 0.48486328125, 1.0498046875, -0.3828125, 0.430908203125, -0.896484375, -0.94482421875, -0.75, -0.56494140625, -0.86865234375, -1.1953125, -0.96923828125, -0.595703125, -0.05810546875, -1.3115234375, -0.010589599609375, -0.5224609375, 1.19140625, -1.0400390625, 0.282958984375, -1.3232421875, -0.167236328125, 0.93994140625, -1.0234375, 1.2275390625, 1.662109375, -0.69775390625, -1.01171875, -0.1531982421875, 0.89892578125, -2.072265625, 0.1654052734375, -1.89453125, 0.034515380859375, -0.144775390625, -0.1939697265625, -0.10394287109375, -1.109375, -0.74755859375, 1.4677734375, -0.64599609375, -0.0171661376953125, 0.541015625, -0.5009765625, 1.4345703125, 1.4951171875, -0.2236328125, -1.0439453125, 1.3447265625, 0.89990234375, 1.689453125, 0.5087890625, 0.0689697265625, -1.9306640625, 0.712890625, 1.1728515625, -0.152099609375, 1.0302734375, -1.333984375, -2.69140625, 0.92333984375, -0.353515625, 0.415771484375, 0.501953125, -1.5791015625, 1.0009765625, 0.068359375, 1.1923828125, -0.55517578125, -0.419189453125, 0.0293731689453125, -0.036163330078125, -1.2880859375, 0.0799560546875, 0.556640625, 0.047332763671875, 1.384765625, 1.2001953125, -0.29833984375, 1.5478515625, -0.52685546875, 0.1312255859375, 0.13134765625, -1.16015625, -0.95654296875, -0.75146484375, -0.2242431640625, -0.69580078125, -0.9423828125, 1.1650390625, -0.147216796875, 0.61328125, -0.294189453125, -0.1771240234375, 0.5693359375, -1.87890625, 0.08465576171875, 0.93310546875, -0.342041015625, -0.982421875, -1.431640625, 0.2139892578125, 0.1458740234375, 0.6279296875, -1.0576171875, 2.28125, -0.134765625, 0.321533203125, -1.9951171875, 0.85888671875, -0.71923828125, 0.02618408203125, 2.375, 0.09686279296875, -0.91162109375, 0.2454833984375, -0.388427734375, 0.042144775390625, -0.1444091796875, 2.0078125, -1.38671875, 1.0576171875, -1.2685546875, 1.0302734375, -1.8583984375, -0.41015625, -0.75146484375, 2.615234375, 0.54833984375, -1.185546875, -0.60693359375, 0.71044921875, -0.58642578125, 0.1607666015625, 0.99853515625, 0.028076171875, 0.1048583984375, -0.2220458984375, 0.188720703125, 1.876953125, -1.0615234375, -1.873046875, 0.08148193359375, 0.08880615234375, 0.828125, -1.5166015625, -0.423828125, 0.6943359375, -1.4013671875, 0.412109375, 0.38427734375, -0.60595703125, 0.62548828125, -0.487548828125, -0.537109375, 1.330078125, -0.3525390625, 1.58203125, 0.72021484375, -0.81884765625, -0.181640625, -0.99267578125, -0.88818359375, -1.7587890625, -0.541015625, 1.0439453125, 0.5732421875, -0.99658203125, -0.250732421875, 0.425537109375, 0.11871337890625, 0.5888671875, 2.00390625, 0.759765625, 0.2685546875, 0.54931640625, 0.5234375, -0.9912109375, -0.274169921875, 1.193359375, -0.642578125, 0.1920166015625, -0.3056640625, 0.53515625, 0.5859375, -0.376953125, -0.053741455078125, -1.8388671875, 0.0263519287109375, -0.367919921875, 0.0712890625, -0.424560546875, -0.77783203125, 0.05169677734375, 1.12109375, 1.400390625, 1.28515625, -1.66015625, -0.01473236083984375, 0.00033402442932128906, 1.27734375, 0.73779296875, -0.346435546875, -1.4267578125, 1.28125, 1.32421875, 0.88427734375, 0.75634765625, -0.8193359375, 0.2408447265625, 0.1710205078125, 0.061279296875, -0.63916015625, -0.619140625, -1.9443359375, 0.2471923828125, 0.205322265625, 0.4091796875, 0.55517578125, 0.07086181640625, 1.59375, -0.0026454925537109375, -1.73046875, -0.7548828125, -0.662109375, 0.94970703125, -0.67822265625, 1.36328125, 2.498046875, -1.197265625, 0.869140625, -0.1673583984375, -0.75634765625, 1.0244140625, 1.7861328125, 1.7890625, -2.509765625, 0.1334228515625, 0.441162109375, -1.5185546875, -0.59423828125, -2.01953125, 1.36328125, 0.332275390625, -0.477294921875, 3.078125, 1.3544921875, -0.477294921875, 0.361572265625, -0.29296875, 0.1552734375, -1.7880859375, 0.93310546875, 2.23828125, 0.415283203125, -0.3876953125, -0.572265625, 1.0263671875, -1.349609375, 0.11175537109375, -1.25, -0.41015625, 0.1947021484375, -0.5126953125, 1.373046875, 0.4951171875, 0.978515625, 1.794921875, -1.4052734375, -0.5615234375, -0.90380859375, 0.279541015625, 0.77197265625, 0.83447265625, 0.397705078125, -1.52734375, 1.01953125, -0.10870361328125, 0.485595703125, 1.501953125, -0.0477294921875, -0.270263671875, -0.921875, -1.3388671875, 1.6953125, -0.41796875, -2.333984375, 0.73486328125, -0.1031494140625, 0.44873046875, 0.1558837890625, 1.193359375, -1.4248046875, -0.6533203125, -0.2900390625, -0.53759765625, 0.60546875, 0.0274200439453125, 1.7470703125, 0.90869140625, 0.00588226318359375, 0.97265625, -0.1240234375, 1.0224609375, -2.04296875, 0.8291015625, -1.4697265625, 0.1767578125, 0.61962890625, 0.19775390625, -0.92138671875, 0.69580078125, 0.5302734375, 1.435546875, -1.115234375, -0.8056640625, 1.84375, 0.303955078125, -1.12890625, 0.45751953125, -0.391845703125, 0.001251220703125, -1.318359375, -1.08984375, -0.5068359375, 0.98095703125, -0.271240234375, -1.8154296875, 1.017578125, 1.1357421875, -0.6513671875, 0.05328369140625, 0.263427734375, -0.771484375, -1.4833984375, 1.619140625, -1.0732421875, -0.456787109375, -0.1475830078125, 0.94921875, -0.9052734375, -0.75244140625, 0.360595703125, 1.5244140625, 0.62939453125, 0.0084991455078125, -0.7998046875, 0.336669921875, 0.2252197265625, 0.07830810546875, -0.36376953125, -2.19921875, -0.17724609375, -0.79736328125, -0.426513671875, 0.0251617431640625, -0.7099609375, -0.40283203125, 0.50537109375, 1.548828125, -1.3525390625, 0.99560546875, -0.59326171875, -0.240966796875, 0.299072265625, 0.71826171875, 0.92041015625, 0.99072265625, 0.4951171875, -1.0537109375, -0.1485595703125, -0.78955078125, -0.12548828125, -1.1904296875, -1.283203125, -0.94140625, 1.2236328125, 0.7412109375, -1.05078125, 0.873046875, -0.312744140625, -0.12060546875, -0.0296173095703125, 1.3837890625, 0.52490234375, 0.705078125, -0.19482421875, -0.7529296875, -1.005859375, 0.098876953125, -0.83251953125, 0.07763671875, -0.32275390625, -1.201171875, -0.2509765625, -0.11376953125, -0.7509765625, 2.478515625, 2.1171875, -0.428955078125, -0.671875, -0.65185546875, -0.1043701171875, 0.499267578125, -3.025390625, 0.55029296875, -0.06524658203125, 0.427490234375, 1.5517578125, -0.082275390625, 0.7373046875, 0.442626953125, -1.47265625, 0.0362548828125, 0.475341796875, 0.300048828125, 1.833984375, 0.27978515625, 0.02374267578125, -0.144287109375, 0.68994140625, -0.315673828125, -0.48583984375, 0.1634521484375, 0.2412109375, 0.51904296875, -1.0458984375, -0.08697509765625, -0.055084228515625, 1.23828125, 0.8935546875, -0.2252197265625, -0.423095703125, 0.91943359375, -0.693359375, 1.7412109375, 0.798828125, -0.52783203125, 0.67822265625, 2.33984375, 0.65478515625, -1.4609375, 0.71142578125, -0.703125, -0.2763671875, -0.5078125, 2.705078125, 0.11895751953125, -0.364013671875, -0.53564453125, -0.71142578125, 0.70849609375, -0.9111328125, -0.50390625, -1.7822265625, -0.98193359375, 0.8505859375, -1.18359375, 0.759765625, 0.978515625, 0.9951171875, -1.3134765625, -0.007572174072265625, -1.7470703125, 0.034393310546875, 0.7314453125, 0.71923828125, 0.98193359375, 0.5810546875, -1.9521484375, -0.6455078125, 0.28662109375, 0.2451171875, -0.1669921875, -0.56640625, -1.5205078125, -1.62109375, 0.37841796875, 0.8154296875, 0.91650390625, -1.22265625, 1.15625, 1.7919921875, -0.91259765625, 1.2705078125, -0.64453125, 0.8818359375, 1.4189453125, 0.880859375, -1.240234375, 0.91064453125, 0.349365234375, 0.35693359375, -1.1376953125, -0.5185546875, -0.70751953125, -1.755859375, -0.646484375, 0.1683349609375, 1.1162109375, 1.1162109375, 0.125732421875, 1.537109375, -0.419189453125, 0.4599609375, 1.4384765625, 1.2626953125, -0.0267181396484375, -0.83447265625, 1.779296875, -1.7607421875, 1.06640625, -0.475341796875, 2.072265625, -0.8037109375, -1.01953125, -0.6884765625, -1.9384765625, -0.105712890625, -0.4873046875, -0.56640625, 0.039764404296875, -0.81005859375, -0.2255859375, -1.56640625, -0.1741943359375, 1.93359375, -0.346923828125, -0.6435546875, -1.1630859375, 0.7978515625, 0.71875, 1.861328125, -1.4267578125, -2.189453125, 0.44970703125, -0.429931640625, -0.51416015625, -0.0267791748046875, -0.4775390625, 0.428466796875, -0.51953125, -1.0673828125, -0.461181640625, 0.84521484375, -0.65087890625, -2.8359375, -1.2744140625, -0.775390625, 0.57080078125, 0.771484375, 0.08087158203125, -0.250244140625, -2.103515625, 1.0087890625, -2.8203125, 0.796875, -0.93896484375, 0.08758544921875, 0.0780029296875, 0.13818359375, 0.224853515625, -3.416015625, -2.595703125, -1.005859375, -0.5341796875, -0.79296875, -0.4580078125, -0.1451416015625, 0.438720703125, 0.0281982421875, -0.1263427734375, 1.1015625, 1.294921875, -1.1455078125, -1.8515625, 1.4033203125, -0.494384765625, 0.12420654296875, -0.39306640625, -2.01171875, -0.299560546875, -0.72607421875, 0.2578125, 1.001953125, 0.440673828125, 0.103515625, -0.228271484375, 0.31884765625, 1.2333984375, -0.468994140625, -0.22412109375, 0.0225372314453125, -0.462890625, 1.8017578125, -1.12890625, -1.6005859375, 1.40234375, 0.1617431640625, -2.09375, 1.8681640625, -0.489013671875, 0.189208984375, -1.5, 0.3623046875, -0.7578125, 0.01983642578125, 0.7373046875, -0.94677734375, -0.87451171875, -0.061431884765625, 0.59521484375, -2.095703125, 0.50830078125, -1.5400390625, 0.9873046875, -0.55810546875, -0.8056640625, 0.1993408203125, 2.4921875, 1.4501953125, -1.0224609375, -1.603515625, 0.6376953125, 0.07562255859375, -1.16015625, 1.109375, -0.113525390625, 1.2802734375, -0.017913818359375, -1.689453125, -0.9677734375, 1.8125, -1.279296875, 0.80029296875, -1.3525390625, 0.42626953125, 0.013031005859375, 2.263671875, 0.927734375, -0.83349609375, 0.330078125, 0.00833892822265625, -0.90380859375, -0.6728515625, -0.37060546875, -2.359375, -1.6259765625, 0.1995849609375, -0.58349609375, -0.89697265625, -1.494140625, -2.115234375, -0.77734375, 0.8564453125, -1.322265625, 1.58984375, -0.7841796875, 2.4140625, 0.488525390625, -1.328125, -0.8427734375, 1.15625, 1.9228515625, -0.7216796875, -2.890625, 0.7861328125, -1.5, 0.31982421875, 2.640625, 0.1566162109375, -0.251953125, -1.509765625, 2.845703125, -0.26220703125, 1.0771484375, 0.7919921875, -0.5625, 0.39111328125, -1.203125, 1.146484375, 1.1298828125, 0.2474365234375, 1.388671875, -1.9609375, 0.1121826171875, -0.40771484375, -2.677734375, 0.262939453125, 0.6328125, -2.78515625, -0.59228515625, 0.143798828125, 0.9189453125, 0.11712646484375, -0.348876953125, 2.17578125, -0.016876220703125, 0.2352294921875, -0.4990234375, -0.301025390625, 1.2822265625, 0.92822265625, 1.1435546875, -1.4814453125, -1.669921875, -0.91015625, -0.83984375, 0.0816650390625, -1.1279296875, -0.28369140625, 0.11102294921875, 1.2470703125, 0.08447265625, 2.4375, -1.427734375, 0.76953125, 0.3056640625, 0.380859375, 0.37646484375, 1.0234375, -0.9814453125, 0.11517333984375, 0.257080078125, -1.2841796875, 1.3779296875, 1.962890625, 0.4580078125, -1.412109375, 0.0017795562744140625, 0.288330078125, -0.3173828125, 0.59228515625, 1.08984375, -0.91357421875, -1.3212890625, 0.358642578125, -0.422607421875, -1.134765625, 0.92529296875, 0.68212890625, 1.5107421875, 1.177734375, -0.55615234375, 0.382080078125, -0.69287109375, 0.296630859375, 0.83203125, -0.30029296875, 1.52734375, 0.5029296875, -0.62451171875, -0.8251953125, 0.4853515625, 1.37109375, 0.4052734375, -1.3408203125, 0.022735595703125, 1.2998046875, 0.65283203125, 0.857421875, 0.9169921875, 0.79248046875, -0.7158203125, 0.5634765625, 0.81005859375, -1.298828125, -1.2373046875, -1.1298828125, 0.6044921875, 0.2464599609375, -2.201171875, 0.4873046875, -0.78955078125, 1.83203125, -1.6201171875, -0.92724609375, 0.9736328125, -2.099609375, -1.55859375, -0.0855712890625, 1.2978515625, -1.1748046875, 1.732421875, 1.6494140625, -0.0038242340087890625, 2.04296875, 0.34326171875, 1.0048828125, 1.4140625, -0.1375732421875, -0.1702880859375, -0.7001953125, 0.438720703125, 1.7841796875, -0.3154296875, -0.190185546875, -0.275634765625, -0.04937744140625, -0.203125, 0.300537109375, -1.1630859375, 2.177734375, -0.62060546875, -0.6904296875, -0.67529296875, -0.01139068603515625, 0.72119140625, 0.66357421875, -0.1005859375, -1.6904296875, -0.497802734375, 0.1328125, -0.09893798828125, -1.2353515625, -1.0703125, -0.79638671875, -1.1552734375, 1.0146484375, 1.0654296875, 0.81591796875, -1.0595703125, -0.93603515625, -0.3984375, -0.947265625, 0.505859375, 0.08282470703125, 0.44482421875, -0.487060546875, 0.208740234375, -1.5732421875, -0.78955078125, -2.802734375, -0.08245849609375, -0.2415771484375, -0.285400390625, -1.517578125, -1.4111328125, -1.265625, -0.650390625, -0.59130859375, 1.0205078125, 0.994140625, -0.6806640625, 0.405517578125, -0.60009765625, -0.314453125, 1.0576171875, 0.5615234375, 0.5703125, -1.8017578125, -1.115234375, -0.14208984375, -0.06854248046875, 2.44921875, 0.60498046875, -0.01297760009765625, 0.07110595703125, 0.25341796875, 0.78271484375, 0.84326171875, -0.69384765625, -0.89990234375, -0.473876953125, 2.140625, -1.9404296875, -0.5693359375, 0.355712890625, 0.57861328125, -0.87353515625, 1.2861328125, -0.57568359375, 0.5341796875, -0.125732421875, 0.119873046875, 2.072265625, -1.947265625, 1.396484375, 1.0107421875, -0.323486328125, 0.5615234375, -0.8974609375, 0.331787109375, -0.57080078125, 0.370849609375, -0.669921875, -1.2646484375, -0.5166015625, 0.466064453125, 0.76904296875, 2.078125, 2.3046875, 0.38037109375, -0.5927734375, 0.1021728515625, 0.1912841796875, -0.814453125, 0.51708984375, -0.6533203125, -0.00179290771484375, -0.07806396484375, 1.0732421875, -0.278076171875, -0.3232421875, -1.51171875, 0.1480712890625, 0.99951171875, 0.78369140625, 0.019287109375, 0.336669921875, -1.384765625, 0.415283203125, 0.92431640625, -0.45458984375, -0.294189453125, 1.9287109375, -0.65185546875, -1.533203125, 2.109375, 0.451416015625, 0.578125, 1.513671875, 0.3779296875, -2.0390625, -0.52783203125, 0.430419921875, 2.029296875, 0.71533203125, 0.70556640625, -0.69140625, -1.041015625, 0.211181640625, 0.235107421875, -0.036712646484375, -1.017578125, -0.88134765625, 0.452880859375, -1.548828125, -0.7138671875, -0.40673828125, 1.15625, -0.62890625, -0.9462890625, 2.013671875, 0.41162109375, 0.5185546875, 0.57177734375, 0.74853515625, 0.48876953125, -0.166748046875, -0.62255859375, 0.2247314453125, -0.32666015625, 0.84228515625, 0.455810546875, -0.58447265625, -2.271484375, -1.7099609375, 2.3671875, 0.490234375, -0.83349609375, -0.82568359375, 1.0068359375, -0.43603515625, -0.83837890625, -1.1201171875, -1.0078125, 0.252685546875, 1.2412109375, -0.387451171875, -2.625, 1.140625, 1.716796875, 0.1434326171875, -0.77197265625, -1.7041015625, -0.8232421875, 0.7607421875, 0.98388671875, -1.2578125, 1.818359375, 0.876953125, -1.23828125, -0.1658935546875, -0.3330078125, 1.3095703125, -0.7568359375, 0.84130859375, 1.0498046875, 0.491943359375, -0.7041015625, -0.501953125, 2.1953125, -2.373046875, -1.375, 0.2027587890625, 0.179931640625, 1.2626953125, 0.0015420913696289062, 0.024658203125, 0.1578369140625, -0.830078125, 0.84716796875, -0.1156005859375, 1.0654296875, -1.2373046875, 1.193359375, -1.68359375, 0.95068359375, -1.4775390625, -0.0119171142578125, -0.52099609375, 0.41796875, 0.478759765625, 0.2783203125, -0.0151214599609375, 0.4990234375, 0.18310546875, -1.0078125, -0.055999755859375, 0.12249755859375, 0.250732421875, -0.23974609375, -0.30029296875, -1.595703125, -0.8525390625, -0.6640625, 0.5078125, -0.076416015625, -0.71484375, 0.5849609375, -0.93212890625, 1.8701171875, -0.0287322998046875, 0.408203125, 0.2144775390625, -0.65087890625, 0.375244140625, -1.359375, -0.1087646484375, -1.2197265625, 1.9228515625, 0.06646728515625, 0.436279296875, 1.232421875, -0.44189453125, 1.662109375, -0.89013671875, 0.78955078125, 2.09375, 0.307373046875, 1.3955078125, 0.541015625, -0.410888671875, -0.10699462890625, 0.12744140625, -1.2568359375, 0.53662109375, -2.02734375, -0.491943359375, -1.517578125, -0.21142578125, 0.47998046875, 0.06378173828125, -0.70263671875, 1.4560546875, -0.062042236328125, 0.446533203125, -0.7412109375, 0.2425537109375, -0.7900390625, -0.76806640625, 1.306640625, -1.8828125, 0.2998046875, -0.184326171875, 0.76708984375, -2.2890625, 1.5400390625, 0.04803466796875, 1.126953125, 0.2257080078125, 0.40283203125, -0.28369140625, -0.19287109375, -0.6484375, 0.1397705078125, -0.80517578125, -0.7314453125, -0.63720703125, 0.99462890625, 0.8349609375, 0.57763671875, 1.125, -0.34423828125, -0.307861328125, 1.2255859375, -0.4775390625, 0.79931640625, -0.06756591796875, -1.3955078125, -1.9892578125, 0.175537109375, 2.373046875, -0.291748046875, -0.40087890625, -1.634765625, 0.7734375, -0.42724609375, 0.459228515625, 0.3896484375, -0.07305908203125, 0.1514892578125, -1.025390625, -0.10992431640625, 1.6171875, -0.1988525390625, 0.93359375, 0.64794921875, -0.87646484375, -0.89794921875, 0.02496337890625, -0.54931640625, 0.47412109375, 0.8369140625, -1.58203125, 0.58203125, -0.9384765625, 0.66748046875, -0.053619384765625, 1.1962890625, 1.6015625, -0.437255859375, 0.9912109375, 0.4130859375, -0.58203125, -2.130859375, -0.2271728515625, 0.2177734375, -0.75927734375, 1.2138671875, -0.955078125, 0.9375, -0.84521484375, 0.398681640625, 0.12127685546875, 0.57763671875, -0.75, -0.489013671875, 0.338623046875, -1.193359375, -0.397705078125, 1.3486328125, 0.6953125, -0.52490234375, -0.294189453125, -0.079833984375, 0.9052734375, -2.068359375, -1.30078125, -1.7001953125, -0.6904296875, -2.56640625, -0.52978515625, 0.59619140625, 1.9970703125, -0.56103515625, -0.68310546875, 1.1455078125, -0.66357421875, 0.493408203125, -0.1683349609375, 0.89453125, 0.02398681640625, 0.9814453125, 0.8466796875, -1.625, -0.204833984375, -0.194580078125, -0.38232421875, 1.0849609375, 0.68798828125, -0.521484375, 1.1591796875, -1.2041015625, 1.5498046875, 0.322509765625, -0.81396484375, 1.1787109375, -0.174560546875, 0.73583984375, 0.2008056640625, 0.744140625, 0.287109375, -0.7392578125, 0.52734375, -0.80029296875, -0.81787109375, 0.98974609375, -0.6044921875, 1.35546875, 0.7568359375, 0.56103515625, -0.224853515625, 0.4873046875, -0.1748046875, -1.2890625, 1.029296875, 0.5302734375, -0.203857421875, 0.67041015625, 0.03411865234375, 1.0751953125, 0.291748046875, 0.004993438720703125, 0.40185546875, 0.90673828125, -0.080078125, -1.70703125, 0.49072265625, 1.9677734375, -0.873046875, 0.152587890625, -0.017181396484375, 0.297607421875, -1.1484375, -0.56396484375, 0.51904296875, -0.7578125, 0.46923828125, -1.1005859375, -0.61279296875, -0.58740234375, 0.47802734375, 0.736328125, -1.869140625, 0.662109375, 2.685546875, 0.7998046875, 0.21826171875, -0.9951171875, -0.2105712890625, 1.0224609375, -0.74658203125, 0.998046875, 0.560546875, -0.525390625, 0.347412109375, 0.2000732421875, -0.74951171875, 0.54541015625, -0.401611328125, -0.6865234375, -0.47265625, 0.0787353515625, -1.837890625, 0.188720703125, -0.247802734375, 0.0848388671875, 0.308837890625, -0.9013671875, -0.4111328125, 0.006153106689453125, -0.1905517578125, 0.92724609375, -0.43212890625, 0.441162109375, 1.083984375, -0.380126953125, -0.1292724609375, 1.232421875, 0.054229736328125, -0.5810546875, -1.037109375, -1.1044921875, 0.32373046875, 0.60400390625, -0.08648681640625, -0.21240234375, 1.212890625, 0.81689453125, -0.58203125, 1.3837890625, -2.07421875, 1.5029296875, 1.82421875, -0.56640625, 1.5380859375, -0.53173828125, 0.349609375, -0.87890625, 3.744140625, -1.3046875, 0.23291015625, -0.48291015625, -1.5615234375, 0.55517578125, 1.6884765625, -0.047454833984375, -0.06475830078125, 0.61767578125, -0.0372314453125, 0.35009765625, 0.3642578125, 0.70556640625, -1.05078125, 2.802734375, 0.78271484375, -0.4345703125, -0.0276336669921875, 1.9228515625, -0.403076171875, -2.130859375, 1.646484375, 2.130859375, 0.70361328125, 0.273681640625, 0.541015625, 1.1484375, 0.270751953125, 0.90966796875, -0.1912841796875, -0.2437744140625, 1.4833984375, -0.736328125, -0.062408447265625, 0.368408203125, -1.13671875, -1.51171875, 0.161376953125, 1.9951171875, 1.5869140625, 0.67626953125, 1.4052734375, -0.256591796875, -0.3662109375, 0.26123046875, 0.751953125, 0.5078125, -0.32568359375, 0.248291015625, -0.708984375, -0.8740234375, -1.4619140625, -0.51611328125, 0.54638671875, 0.2431640625, 0.2098388671875, 0.115478515625, 0.6494140625, -0.388427734375, 1.26953125, 1.724609375, 0.365234375, -0.8896484375, 0.2529296875, 0.163330078125, 1.1494140625, -1.1796875, -1.9375, 1.140625, -1.361328125, 0.478515625, -0.2587890625, 0.546875, -1.07421875, -1.5791015625, -0.52685546875, 1.5771484375, -0.71435546875, 0.25732421875, -0.8466796875, 0.435302734375, 0.81396484375, -1.1064453125, 1.12890625, -0.72802734375, -0.406982421875, 0.361083984375, 0.487548828125, 0.583984375, 0.07373046875, -0.33447265625, 1.126953125, 0.61962890625, 0.368896484375, -1.2607421875, 2.146484375, -0.7939453125, -0.87939453125, 0.90380859375, 0.224609375, 0.69189453125, -2.4140625, 0.498779296875, -0.72412109375, 1.44140625, -1.4560546875, 1.0029296875, -1.5556640625, -0.2122802734375, 0.7890625, 0.0035343170166015625, -0.70068359375, -0.39697265625, 0.468994140625, -0.6884765625, -0.0848388671875, -0.1768798828125, 0.4130859375, 0.9423828125, 0.1043701171875, -1.9716796875, 2.8828125, 0.342041015625, 1.509765625, 0.62255859375, 1.166015625, -0.57470703125, -0.01025390625, -0.64208984375, -0.67529296875, 0.0435791015625, 0.39990234375, 0.8427734375, 0.79736328125, 0.65478515625, -0.26904296875, 0.260009765625, -0.464599609375, -0.8369140625, -0.423095703125, 0.367919921875, 0.154541015625, -0.34326171875, 0.64306640625, 0.66455078125, 0.489501953125, -2.265625, -0.2244873046875, 0.96728515625, 0.061737060546875, 1.3564453125, 1.0029296875, 0.2021484375, -0.311767578125, 0.76416015625, -0.353759765625, 1.8818359375, -1.669921875, -0.363525390625, -0.79443359375, -0.5322265625, 2.630859375, -1.748046875, -0.1112060546875, -0.90625, -0.484130859375, -1.4052734375, 0.40234375, -1.66015625, 2.181640625, 0.1318359375, -1.615234375, 1.2021484375, -1.1083984375, 0.369873046875, 0.81982421875, -0.2210693359375, 1.783203125, 0.056915283203125, -0.2142333984375, 1.1337890625, -0.314453125, -0.264892578125, -1.6640625, 0.364990234375, -0.431640625, -0.033538818359375, 1.048828125, -0.436279296875, -0.6015625, -0.96630859375, 0.45556640625, -0.2056884765625, -2.099609375, -0.78759765625, -0.53369140625, -1.296875, 0.65478515625, 1.17578125, 0.64501953125, 1.220703125, -0.7412109375, -1.728515625, 0.87158203125, 0.49267578125, -0.480224609375, 0.150146484375, -0.0589599609375, -0.43896484375, -0.247802734375, 0.0762939453125, -0.076416015625, -2.044921875, 1.0546875, 0.1112060546875, -0.01104736328125, 0.75, -0.58447265625, -0.32470703125, -1.01171875, -0.448486328125, 0.1761474609375, -0.1470947265625, 1.501953125, -0.720703125, 0.12469482421875, 0.1435546875, -0.58740234375, 0.03033447265625, 1.03125, -0.039947509765625, -0.4052734375, -0.08551025390625, -0.76904296875, -0.368896484375, -0.67333984375, -0.44873046875, 1.2763671875, 0.533203125, 0.3896484375, 1.83984375, -1.1923828125, 0.873046875, -0.326171875, -0.6904296875, 0.14990234375, -0.29443359375, 0.7861328125, -2.25390625, 1.27734375, 1.2060546875, -0.9482421875, 0.9111328125, 0.8369140625, -1.064453125, 0.014617919921875, 0.419677734375, 2.455078125, 0.5703125, -0.751953125, 1.2587890625, -0.09967041015625, -1.0947265625, -1.2109375, 0.1961669921875, 0.9677734375, -0.8251953125, 0.947265625, -0.69970703125, 0.7138671875, 0.40966796875, -0.63818359375, -1.33203125, -0.6162109375, -1.03515625, -1.177734375, 0.1005859375, -0.73095703125, -0.9541015625, 1.01171875, 1.390625, -0.87451171875, -0.1995849609375, -0.6240234375, -0.560546875, 1.1279296875, -0.343017578125, -1.9697265625, -0.03778076171875, -2.337890625, 0.00826263427734375, -0.1455078125, 0.391845703125, 0.47412109375, -0.307373046875, 0.74267578125, -0.30712890625, -0.52001953125, -0.4130859375, -0.416748046875, 1.169921875, -1.7001953125, -0.309326171875, 0.2235107421875, -0.83935546875, -0.89111328125, 0.69677734375, -0.654296875, 0.234375, -0.99169921875, -0.344482421875, -1.4306640625, -0.082763671875, 1.310546875, -1.640625, -2.31640625, 1.1240234375, -0.053192138671875, 0.0168609619140625, -0.29443359375, 1.3916015625, 2.056640625, -0.96826171875, -0.6142578125, -0.53955078125, -0.60595703125, 0.234375, -0.03277587890625, 1.0712890625, 0.61328125, 0.51953125, 2.380859375, -1.1337890625, -0.2437744140625, -1.005859375, -0.55712890625, -1.05078125, 0.31201171875, 0.267822265625, -0.9775390625, 0.144775390625, 0.47607421875, -0.10186767578125, -0.54052734375, -1.525390625, -0.51025390625, 1.02734375, 0.451171875, -0.495849609375, -0.426513671875, -0.5615234375, 0.9599609375, 0.316650390625, 0.0587158203125, -0.7314453125, -1.189453125, 1.228515625, 0.6875, 2.732421875, -0.06146240234375, 0.720703125, 0.1336669921875, -0.1546630859375, -0.444091796875, 1.2080078125, 1.4794921875, -1.134765625, -0.04669189453125, -0.6748046875, 1.7431640625, 0.28857421875, -0.1646728515625, -0.30029296875, -0.66064453125, -0.375732421875, 0.55810546875, -0.00902557373046875, -0.81982421875, -0.0902099609375, -1.1552734375, -1.4326171875, -0.9140625, 0.1636962890625, 1.21484375, 0.224609375, 1.2705078125, 0.8916015625, 0.5009765625, 2.267578125, -0.461181640625, -0.489013671875, 1.138671875, -1.0478515625, 0.32177734375, -0.53515625, -1.0185546875, -0.277587890625, -0.845703125, -0.75390625, -1.28515625, 1.544921875, -1.1484375, -0.09716796875, 2.9609375, 0.81640625, -0.99755859375, -0.76416015625, 0.09136962890625, 0.54541015625, -0.69921875, 1.1513671875, -0.65234375, 0.5478515625, 0.14599609375, -0.52197265625, 1.2099609375, 0.7744140625, 0.001556396484375, 0.0279083251953125, -2.158203125, 0.338134765625, -0.61083984375, -0.431396484375, 0.0171661376953125, -1.009765625, 0.78173828125, -0.58154296875, 0.82958984375, -0.111328125, 0.01490020751953125, -0.93212890625, -0.216064453125, -0.10498046875, -0.82861328125, -0.1138916015625, -0.288330078125, -0.71923828125, 0.374267578125, -0.91943359375, 1.39453125, -0.241455078125, 1.3076171875, -0.546875, -1.134765625, -1.0869140625, -0.2822265625, -0.72119140625, 1.9970703125, -1.513671875, 0.59130859375, 0.4091796875, 0.158203125, 2.21875, -0.54931640625, 0.38427734375, 1.173828125, -0.1820068359375, 0.068603515625, -0.16796875, -0.032867431640625, -0.056793212890625, 0.20263671875, -0.91015625, 0.85498046875, 0.464599609375, 0.1981201171875, -0.369873046875, 1.3046875, 0.89892578125, 0.67724609375, -0.293212890625, 1.52734375, 0.36572265625, -0.54052734375, -1.9423828125, 1.0439453125, 0.61328125, -0.046905517578125, -0.9873046875, 0.31494140625, 1.8583984375, 0.5, 0.10858154296875, 0.479248046875, -0.84912109375, -0.0238494873046875, 1.4130859375, 0.038970947265625, 1.2548828125, -0.254150390625, 1.2939453125, 0.57177734375, -0.260009765625, -1.0107421875, -0.4619140625, -1.005859375, 0.90283203125, -0.4384765625, 1.2421875, -2.025390625, -0.71484375, 1.1640625, 2.34375, 1.16796875, 0.046142578125, -0.5810546875, 0.48095703125, -0.376953125, 0.267333984375, 0.0518798828125, 0.8447265625, 0.6298828125, 0.544921875, 0.046295166015625, 0.70556640625, -0.76171875, 0.83642578125, 1.2109375, 0.398681640625, -2.421875, -1.486328125, 0.75244140625, -1.0126953125, -2.029296875, -1.1484375, -0.282470703125, -0.5478515625, -1.6884765625, -0.0158538818359375, 0.81982421875, 0.6103515625, -0.2529296875, 0.75830078125, -0.9580078125, 0.69873046875, 0.47314453125, 0.38916015625, 0.1983642578125, -1.798828125, -1.6962890625, -2.244140625, -0.63671875, 0.125, -1.3056640625, -0.109130859375, -1.21875, -0.289794921875, 0.45703125, 0.5771484375, 0.8525390625, 0.06903076171875, -0.6611328125, -1.42578125, -0.65380859375, -1.0732421875, 2.490234375, -2.892578125, -0.204833984375, 0.58837890625, -0.5, -1.5146484375, 0.347412109375, -0.0175628662109375, 0.467529296875, 0.212158203125, -0.158447265625, -1.333984375, -0.61572265625, -0.1715087890625, 1.1025390625, -1.306640625, 1.6728515625, 0.1553955078125, -0.2332763671875, -1.2314453125, 1.6328125, 0.270263671875, -0.62353515625, -1.02734375, 0.1973876953125, -0.6611328125, -0.544921875, -0.2061767578125, -0.619140625, 0.50341796875, -1.7314453125, 1.0751953125, 0.40234375, -0.5009765625, 0.00016200542449951172, -1.5078125, -0.56396484375, 0.0191192626953125, 0.76318359375, 0.94873046875, 1.1767578125, 0.43896484375, 1.486328125, -0.2034912109375, 0.0985107421875, -0.382568359375, 0.206298828125, -0.373046875, -2.125, -0.08770751953125, 0.8583984375, -0.95361328125, 1.0810546875, 0.55224609375, 0.373779296875, 0.748046875, -1.0048828125, 0.84765625, 0.86328125, 0.515625, -1.7392578125, -0.7392578125, 1.6552734375, -0.279541015625, 1.306640625, 0.6005859375, 1.5888671875, 0.64892578125, 0.432861328125, 2.279296875, 0.1641845703125, 0.436279296875, 1.9033203125, -0.947265625, -1.5517578125, 0.97314453125, -0.11163330078125, -0.6640625, -0.192626953125, 1.03515625, 0.6591796875, 0.060302734375, 1.0263671875, 1.1123046875, 0.32080078125, -0.91162109375, 0.75537109375, 1.9990234375, -3.15625, 0.02569580078125, -0.355224609375, 1.3427734375, 0.04461669921875, 0.861328125, 0.67333984375, 1.1279296875, 0.14306640625, -0.82861328125, 0.9306640625, -1.6865234375, -2.103515625, 1.1474609375, -0.1986083984375, 0.59375, 1.39453125, -0.51220703125, -1.2001953125, 0.29736328125, 0.1929931640625, -0.380615234375, -0.55908203125, -0.389404296875, -0.51708984375, -0.287353515625, 0.802734375, -0.77392578125, -0.034759521484375, 0.71630859375, 1.158203125, -0.307861328125, 0.8115234375, 0.275146484375, -0.2479248046875, 1.8564453125, 1.0986328125, 0.1849365234375, -0.71337890625, -0.3671875, -0.2139892578125, -0.78515625, 1.0556640625, -0.450439453125, -0.09796142578125, -0.214111328125, 0.9033203125, -1.6875, 0.9306640625, 0.30322265625, 1.4833984375, 1.55078125, 1.744140625, -1.578125, -0.5634765625, -0.64990234375, 1.40234375, -1.6318359375, -1.099609375, -0.6962890625, -1.0263671875, 1.548828125, 0.57958984375, -0.3818359375, 1.041015625, 0.0980224609375, -1.0849609375, -0.289306640625, -0.311767578125, 0.81884765625, 0.1533203125, -1.083984375, 0.028656005859375, -0.64697265625, -0.72314453125, -2.095703125, -1.5322265625, 0.59130859375, -1.80078125, 0.25, 0.8134765625, 0.070068359375, 2.37890625, 1.31640625, 2.109375, 0.73046875, 2.51953125, 1.439453125, 0.26220703125, 1.365234375, -0.479736328125, -0.92138671875, 0.7685546875, 0.80224609375, 1.5390625, -0.95361328125, -0.59521484375, -0.8681640625, -1.4462890625, -0.349609375, -1.8525390625, -0.8427734375, 0.6689453125, -1.4375, -0.1484375, -1.44921875, 0.72607421875, 1.6533203125, 1.658203125, -0.4052734375, -0.68994140625, 0.0258331298828125, 0.397705078125, -0.60986328125, 0.52294921875, 0.71923828125, 0.69775390625, 0.86572265625, 0.6259765625, -3.01953125, -2.40625, -0.361572265625, 0.90478515625, -0.322265625, 1.0263671875, 0.495849609375, -0.465087890625, -0.4853515625, 0.79736328125, 1.837890625, 1.7763671875, -0.51220703125, -0.6455078125, 0.96875, -1.46484375, 0.0704345703125, -1.0400390625, -0.03790283203125, -0.88916015625, -0.160400390625, 1.2431640625, -1.7021484375, -0.642578125, -0.66064453125, -1.455078125, 0.607421875, -0.89599609375, -0.73388671875, -1.6142578125, 0.341552734375, 0.1923828125, -0.1552734375, -1.5849609375, 1.361328125, 0.8837890625, 1.083984375, 0.59912109375, -0.47265625, 0.234375, -0.7431640625, 0.95263671875, -2.826171875, 1.2548828125, 0.480224609375, 0.9912109375, 0.2322998046875, 1.017578125, -0.31494140625, 0.62109375, 0.09246826171875, -0.0095367431640625, -0.59130859375, -1.404296875, 0.81787109375, -1.33984375, -0.473388671875, 0.56591796875, -0.11578369140625, 0.404296875, -1.2568359375, 0.63330078125, -1.1201171875, 0.35107421875, 0.166259765625, 1.576171875, -1.1748046875, 0.5537109375, -0.041961669921875, 2.69140625, 0.2373046875, 0.89208984375, -0.1331787109375, -0.25341796875, -0.77099609375, -0.07080078125, 0.10113525390625, -0.1646728515625, -1.66015625, 0.17431640625, -0.301513671875, -0.921875, -1.560546875, -1.3525390625, 1.1923828125, 0.578125, -1.119140625, 0.32666015625, 0.1739501953125, 1.853515625, -0.01493072509765625, -0.345947265625, 0.51513671875, -0.88720703125, 0.048919677734375, -0.361572265625, 0.08648681640625, -0.3125, 1.4443359375, -0.3623046875, -0.7158203125, 1.9326171875, 0.1390380859375, -1.93359375, 0.86181640625, -0.409912109375, -0.281494140625, 0.53125, -0.069580078125, -1.3759765625, 1.1474609375, 0.384521484375, 0.4951171875, 0.041015625, 0.93896484375, -0.8583984375, 0.487060546875, -0.61669921875, -1.416015625, 0.60546875, 0.083984375, -0.56298828125, 0.501953125, 0.3916015625, 0.2086181640625, -0.415283203125, -0.142822265625, -0.4453125, -0.63623046875, -2.154296875, -0.2017822265625, 1.54296875, 0.482177734375, -0.0880126953125, -0.14501953125, 0.814453125, -0.67822265625, 0.92822265625, 0.2398681640625, 0.3447265625, 0.326904296875, -0.72021484375, 0.591796875, 0.8994140625, -0.094970703125, -0.2470703125, 2.474609375, -0.310302734375, 0.8154296875, 0.5625, -1.8779296875, 0.037933349609375, 0.68017578125, -1.4853515625, 0.52099609375, 0.3603515625, 0.1900634765625, -0.49609375, -1.0517578125, 0.67919921875, 0.9052734375, 1.162109375, 0.705078125, -0.334228515625, 0.75830078125, -0.34033203125, -0.8701171875, 0.79833984375, -0.97412109375, -0.1815185546875, -1.3779296875, -0.81396484375, -1.03515625, -0.2120361328125, 0.54833984375, 1.1435546875, 0.388916015625, 0.80517578125, -2.43359375, 1.8583984375, 0.29443359375, 0.40869140625, -0.1641845703125, -1.1552734375, -0.31982421875, 0.1806640625, -0.0960693359375, -0.384521484375, -0.0408935546875, 1.35546875, -1.8095703125, -1.013671875, -0.63427734375, -1.560546875, 0.359375, -1.7685546875, -1.3154296875, -0.54833984375, 1.0048828125, 0.2049560546875, -0.360107421875, -0.43310546875, -1.4033203125, 0.316650390625, 0.2176513671875, 1.7509765625, 1.97265625, 1.197265625, -0.108154296875, -0.01517486572265625, 0.88671875, -0.654296875, 1.12890625, -0.701171875, -0.36669921875, 1.8916015625, -0.744140625, 1.3203125, -1.5185546875, -1.189453125, 0.88720703125, 1.4326171875, -0.263916015625, -0.6396484375, -0.0316162109375, -0.0631103515625, -0.281982421875, 1.3681640625, -0.456298828125, -0.72216796875, 1.1591796875, 0.203857421875, 0.08258056640625, 1.2353515625, 0.7119140625, -0.693359375, 0.89306640625, 0.7177734375, 0.82763671875, -1.7724609375, -0.5078125, 0.1458740234375, 2.857421875, -1.076171875, 0.31494140625, 0.003742218017578125, -0.468994140625, -0.08544921875, -1.466796875, -0.68994140625, 0.85986328125, -0.705078125, 1.90234375, 0.68701171875, 1.75390625, 1.83203125, -0.85498046875, -1.109375, 1.19921875, -0.143310546875, -0.796875, 1.4765625, -0.27490234375, -0.51953125, -1.8720703125, -1.3681640625, 0.0936279296875, -1.248046875, -0.1829833984375, -0.71044921875, 1.4287109375, -0.372314453125, -0.6279296875, 1.478515625, -0.7548828125, -1.0908203125, 0.325927734375, -0.92138671875, -0.53076171875, -0.282958984375, -0.0230865478515625, 0.60107421875, 1.55078125, 1.5166015625, -0.7216796875, -2.580078125, -0.822265625, -0.8583984375, 0.22607421875, -0.583984375, -1.2509765625, -1.24609375, -0.53955078125, 0.328857421875, 0.65966796875, 0.1968994140625, -0.62255859375, 0.84130859375, 1.9931640625, -0.150146484375, -0.83056640625, 0.646484375, 0.7998046875, -0.1793212890625, 0.4228515625, 0.8740234375, 0.2093505859375, -0.44189453125, 0.341796875, 0.82861328125, 0.0187225341796875, 0.57177734375, 0.8427734375, -0.2225341796875, -0.47705078125, 1.224609375, 1.5703125, -0.015869140625, 1.853515625, 1.98046875, -0.53076171875, 2.333984375, -0.4951171875, 1.498046875, 0.84814453125, 1.2392578125, -0.359130859375, 0.70703125, 2.046875, -0.2088623046875, -0.339111328125, 2.783203125, -0.393798828125, 0.19873046875, -2.150390625, 0.146728515625, 0.256103515625, 1.7509765625, -1.21875, 0.291259765625, -0.681640625, -1.2412109375, -0.0714111328125, -1.302734375, -1.23046875, 0.13720703125, 0.15185546875, 1.0234375, 1.189453125, 0.470703125, 0.6484375, 0.5244140625, 1.4052734375, -0.47412109375, -2.259765625, 0.572265625, 0.7451171875, -0.6318359375, 1.5927734375, -0.4130859375, -0.100341796875, 0.287109375, 0.3935546875, -0.384521484375, 1.7744140625, -2.033203125, 0.08935546875, 0.05108642578125, 0.37109375, -1.046875, 0.82568359375, -2.1015625, 0.57080078125, -1.1533203125, 0.270751953125, 0.87744140625, -0.8359375, 1.79296875, 0.77294921875, -0.0909423828125, 0.78271484375, 0.61376953125, -0.288818359375, 0.73779296875, 0.5205078125, 1.552734375, -1.392578125, 1.21484375, 0.6328125, -0.451171875, 0.41162109375, -0.5341796875, 1.6650390625, 2.291015625, -0.8740234375, 0.64306640625, 1.7861328125, -0.9287109375, -0.86669921875, -0.64599609375, 0.2076416015625, 0.75537109375, 0.736328125, 1.0, 1.12890625, -1.873046875, -1.826171875, 1.59375, -0.8505859375, 1.345703125, 0.771484375, -1.390625, 0.853515625, 0.29052734375, -0.43017578125, -0.5107421875, 0.53173828125, 1.8662109375, -0.8212890625, -0.52783203125, 0.8720703125, -0.748046875, -0.6552734375, -3.408203125, -0.56494140625, 0.48876953125, -0.377685546875, -0.2183837890625, -0.71435546875, -0.3916015625, 0.1468505859375, 0.1240234375, -1.046875, -1.2158203125, -0.01287078857421875, -1.177734375, -1.3837890625, 1.02734375, -0.76025390625, 0.490234375, -0.529296875, -0.81689453125, -1.2529296875, 0.4072265625, -0.294921875, 0.121826171875, -0.70361328125, 0.80029296875, -2.2578125, -0.397705078125, 1.076171875, -0.57666015625, 0.6279296875, 2.12109375, 0.299560546875, 2.150390625, 0.669921875, -0.900390625, 0.1485595703125, -0.0095672607421875, 0.8828125, -0.669921875, -0.47998046875, -0.3671875, 1.3681640625, -1.1328125, -0.018035888671875, 0.030853271484375, 1.7451171875, -1.923828125, 0.30810546875, -1.365234375, 0.52294921875, -0.642578125, -0.9677734375, -0.66748046875, 0.58984375, -0.75244140625, 0.50390625, 0.73388671875, 0.8984375, -1.6884765625, -0.556640625, 0.01381683349609375, 0.95068359375, 1.3173828125, -1.0849609375, 0.5810546875, 0.88330078125, -0.1187744140625, 0.4296875, -0.1978759765625, -1.173828125, 0.40283203125, -0.2200927734375, 1.4384765625, -2.16015625, 0.6552734375, 1.015625, 0.958984375, 0.07147216796875, 0.75927734375, 1.232421875, -1.7939453125, 0.029449462890625, 0.24462890625, 1.75390625, 1.1650390625, -0.54736328125, 0.52490234375, 1.591796875, 0.2666015625, -0.88330078125, -0.08905029296875, -0.3076171875, 0.028076171875, 0.615234375, 0.9609375, -1.998046875, -0.75048828125, 1.1318359375, 1.3046875, -0.09259033203125, -0.658203125, 0.57373046875, 0.326904296875, 0.251953125, 0.11029052734375, -0.6103515625, -0.143798828125, 1.94921875, -1.970703125, -1.0654296875, -1.0029296875, 0.004337310791015625, 0.81005859375, -0.39111328125, -0.82470703125, 0.356201171875, 0.054779052734375, -0.7939453125, 1.98046875, -0.990234375, 0.1492919921875, 0.25390625, 1.876953125, -1.435546875, 0.62646484375, -1.0390625, -0.33349609375, 1.4833984375, 0.19091796875, 1.4248046875, 1.1806640625, 0.82177734375, 2.171875, -0.260009765625, -0.53125, -0.431396484375, -0.248779296875, -0.4658203125, -0.07403564453125, 0.97705078125, 0.06549072265625, -0.284423828125, -1.255859375, 0.67822265625, 3.0859375, -1.4150390625, 0.1427001953125, 1.44921875, -0.456787109375, -0.8232421875, 0.97314453125, -1.1044921875, -1.8271484375, 0.274169921875, -1.01953125, 0.291015625, 0.3896484375, 0.053314208984375, -0.9365234375, -1.1044921875, 2.134765625, 0.032257080078125, -0.408447265625, 0.34619140625, -0.759765625, -0.260009765625, -0.291259765625, -1.6025390625, -0.445556640625, 1.3798828125, 0.5478515625, 0.390869140625, 0.78466796875, -0.218505859375, -1.3935546875, 1.4970703125, -1.6484375, -0.1280517578125, 0.2379150390625, -1.3876953125, 0.2110595703125, 1.0751953125, -1.0810546875, 0.2010498046875, 2.544921875, -1.0048828125, 0.3134765625, 0.11474609375, 0.039581298828125, 0.71484375, 0.03961181640625, 0.404296875, -0.77099609375, -0.353759765625, -1.7080078125, 1.0625, -1.224609375, -0.1781005859375, 0.55419921875, 0.347900390625, 0.2978515625, -1.2958984375, 1.2705078125, -1.5048828125, 2.005859375, 1.8369140625, 0.5576171875, 0.306640625, 0.400146484375, -0.53759765625, -0.44189453125, 1.0869140625, -0.12939453125, 0.424072265625, 1.6904296875, 0.6484375, -0.9560546875, 0.6923828125, 0.904296875, 0.11248779296875, -0.2259521484375, -0.4990234375, 0.71435546875, 0.953125, 0.82958984375, -1.5068359375, 0.45849609375, -0.6357421875, 0.4306640625, 0.82861328125, -0.306396484375, 0.70068359375, 0.56201171875, 1.2431640625, -0.611328125, -1.4423828125, 0.99169921875, 1.9560546875, 0.64111328125, 0.89208984375, -0.037445068359375, -0.0811767578125, -1.146484375, 2.2109375, -1.7568359375, 2.12890625, 0.619140625, 1.1953125, -1.7119140625, -1.0224609375, 0.72412109375, 1.2822265625, -0.9990234375, -1.8798828125, -0.90478515625, 0.57958984375, -1.1259765625, 1.0556640625, 1.32421875, 0.57568359375, 1.2626953125, -0.217041015625, -1.2021484375, 0.8427734375, -1.296875, -0.74169921875, 0.0262451171875, 0.76708984375, 0.6748046875, 0.225341796875, -1.46484375, 2.5078125, -0.5625, 1.2998046875, 1.017578125, 0.81787109375, -1.03515625, -1.3447265625, -0.1717529296875, -0.396484375, 0.31396484375, 1.9404296875, 1.037109375, 0.697265625, -0.374267578125, 1.1474609375, 1.197265625, -0.72119140625, -0.63037109375, -0.673828125, 0.53955078125, -0.31005859375, 1.232421875, -1.7470703125, 0.95263671875, -0.607421875, -0.54052734375, -0.393798828125, 1.75390625, -2.31640625, -1.380859375, 0.12445068359375, -0.323486328125, -0.04205322265625, 0.334228515625, 1.021484375, 0.20068359375, 0.11199951171875, 1.3857421875, -1.076171875, -2.001953125, 1.2802734375, -0.99072265625, -1.0546875, -0.26708984375, 0.6875, -0.998046875, 1.8525390625, 0.65673828125, -0.323974609375, 1.095703125, -0.63330078125, 0.99755859375, 1.0859375, 1.107421875, 0.96337890625, 0.69873046875, 0.88916015625, 0.70263671875, 1.404296875, -0.298095703125, 0.951171875, 0.218505859375, -1.150390625, 0.55810546875, 0.433837890625, 0.55322265625, 1.3994140625, 0.748046875, 1.59765625, -0.84228515625, 1.4248046875, 0.82568359375, -1.7431640625, 1.05859375, 0.6767578125, -0.0584716796875, 0.568359375, 1.107421875, 1.3154296875, 0.740234375, -0.186767578125, 0.034332275390625, 0.3603515625, 0.533203125, 0.771484375, -0.2137451171875, -0.2432861328125, -0.0167236328125, 0.728515625, 1.1787109375, 0.5888671875, 1.12890625, -0.11236572265625, 0.275390625, 0.966796875, -0.048095703125, 0.693359375, -0.57861328125, -0.46044921875, -0.1982421875, -0.91162109375, -0.026885986328125, -1.12109375, -0.7177734375, -0.97216796875, 1.8115234375, 0.373046875, 0.351318359375, 1.01953125, 0.1614990234375, -2.232421875, 0.256591796875, 0.0165252685546875, 0.4423828125, -0.82080078125, 0.99169921875, 0.04193115234375, -0.86083984375, 0.046875, -0.99951171875, 0.1890869140625, -0.3349609375, -0.830078125, 1.55078125, 0.68212890625, 1.2763671875, -0.365234375, -0.0124359130859375, -0.751953125, -0.66650390625, 0.466796875, -0.69189453125, -0.28955078125, 0.75732421875, -0.7626953125, 0.9921875, -1.7626953125, 1.2451171875, -1.439453125, -0.0295562744140625, 0.63623046875, 0.2445068359375, -0.85107421875, 2.53125, -0.2529296875, 0.469970703125, -0.81787109375, 0.263427734375, 1.107421875, -1.228515625, -1.7392578125, 1.0791015625, 1.4814453125, 0.335693359375, -1.248046875, -0.439453125, 1.7001953125, -1.77734375, -0.85693359375, 0.393798828125, 3.123046875, -0.1732177734375, -0.0994873046875, 2.6953125, -0.9169921875, 0.45458984375, -0.3837890625, -0.43017578125, -0.81298828125, 1.4189453125, 0.00652313232421875, 0.08343505859375, -1.2841796875, 0.318115234375, -1.2158203125, 0.1348876953125, -1.287109375, 0.5224609375, -0.302978515625, -0.7294921875, -1.3330078125, 0.47314453125, -1.2998046875, -1.099609375, -0.40087890625, -0.467529296875, -2.109375, -0.3173828125, 1.4716796875, -0.16455078125, -2.66796875, 0.1505126953125, 0.29833984375, 1.3818359375, 0.0291595458984375, 0.16162109375, -0.5732421875, -0.062408447265625, 0.27880859375, 0.3525390625, 0.67333984375, 3.419921875, -0.07293701171875, 0.07257080078125, 0.7392578125, 0.19580078125, -2.3515625, 0.2344970703125, -0.9912109375, -1.458984375, -0.107421875, 0.10797119140625, -1.884765625, 0.5146484375, 1.044921875, -1.03515625, 1.4326171875, -0.25146484375, -0.1861572265625, 0.80126953125, 2.125, 0.48681640625, -1.126953125, 1.3720703125, -0.333251953125, 0.34228515625, -2.126953125, -0.7900390625, 0.93701171875, 0.53173828125, 0.269287109375, -0.1700439453125, 0.6767578125, -2.23828125, 0.94970703125, -0.437744140625, 0.04107666015625, -1.134765625, 1.33984375, 0.6796875, 0.262451171875, 0.66552734375, -1.7431640625, 1.2392578125, -0.74365234375, -1.2041015625, -0.051605224609375, 0.3251953125, -1.515625, -0.367431640625, 1.45703125, 0.12139892578125, 0.1390380859375, 0.9365234375, 0.0955810546875, 0.69580078125, 0.56005859375, 0.52587890625, 0.4482421875, 0.57275390625, 0.63525390625, 1.71875, -0.1822509765625, 0.002674102783203125, 1.20703125, -0.626953125, 0.78564453125, 2.12890625, 0.126220703125, 0.40771484375, 1.4326171875, 0.5029296875, 2.369140625, -0.8837890625, -0.1385498046875, -1.267578125, -1.6875, -0.1229248046875, 0.4541015625, -0.048492431640625, 1.0634765625, -1.0, 0.004489898681640625, 0.9072265625, 0.5966796875, 0.6279296875, 0.243896484375, -1.51953125, -0.896484375, 1.4658203125, 1.4609375, 1.177734375, -0.6259765625, -0.1134033203125, 0.0924072265625, -0.55712890625, 1.138671875, -0.10345458984375, -0.73046875, -0.159912109375, -1.5458984375, -0.46484375, 0.5830078125, -0.26904296875, 0.12152099609375, -1.3466796875, 0.6416015625, 1.15625, 1.505859375, 0.25634765625, -1.880859375, -1.0673828125, 2.521484375, -1.0986328125, -0.69189453125, 0.39111328125, 0.3603515625, -1.029296875, -0.72216796875, 0.45556640625, -0.056427001953125, 0.861328125, -0.615234375, -0.466064453125, 0.55078125, -1.03515625, 1.3876953125, 0.046478271484375, -0.0005130767822265625, 1.1484375, -0.77783203125, -1.1787109375, 0.472412109375, -0.175048828125, 1.2197265625, 0.39453125, -1.4736328125, 1.5654296875, 0.07421875, 0.455078125, -1.2939453125, 0.419677734375, 0.955078125, 0.455078125, 2.0703125, 0.1112060546875, 0.483642578125, -0.646484375, -1.576171875, -1.0908203125, -1.794921875, 1.9111328125, -0.12890625, 0.4892578125, -0.385498046875, 0.5390625, -0.98583984375, -1.2021484375, -0.78369140625, 1.6416015625, 0.08087158203125, 1.6201171875, -0.90185546875, -2.203125, 2.30859375, 1.2373046875, -0.87841796875, 0.62890625, 1.1953125, -1.0380859375, 0.56787109375, 0.87744140625, 1.65234375, -1.6103515625, 0.49365234375, 1.177734375, 0.66943359375, -1.0830078125, 1.259765625, 0.35986328125, 0.72119140625, 0.332275390625, 2.46875, -0.5966796875, -0.31396484375, -0.6826171875, -0.92822265625, -0.75439453125, -1.0322265625, 0.51708984375, -0.258056640625, 0.24365234375, -0.6240234375, 2.1796875, 1.2626953125, 0.079345703125, -0.89892578125, -0.028167724609375, 0.5947265625, -0.71484375, 1.3134765625, -0.1494140625, 0.9931640625, -0.83544921875, -0.8271484375, 0.1114501953125, 0.058837890625, -1.166015625, -0.677734375, 0.322265625, -0.10748291015625, 0.88623046875, 1.9033203125, -0.1956787109375, 0.415771484375, -1.900390625, -0.8896484375, 2.62890625, -1.0126953125, 0.62158203125, 0.80224609375, 1.3017578125, -1.0576171875, 1.0615234375, -0.241943359375, -1.1787109375, -3.107421875, -1.939453125, 0.363525390625, -0.0501708984375, 1.0166015625, -0.544921875, 0.151123046875, 0.74755859375, -1.0693359375, 1.9638671875, 0.81103515625, -0.333984375, 0.2342529296875, -0.019683837890625, 1.046875, 0.8994140625, 1.7978515625, 1.20703125, 0.74658203125, -0.0151519775390625, -0.7236328125, -0.7080078125, 1.3359375, 0.55126953125, -0.68505859375, -0.74755859375, -1.3388671875, -0.7177734375, -2.294921875, 0.10003662109375, -1.8134765625, -0.53759765625, 2.328125, -0.2066650390625, -0.880859375, 0.5478515625, 0.1370849609375, -0.46240234375, 1.0693359375, -1.193359375, -0.69091796875, -0.389892578125, 2.33203125, 0.6171875, -1.1953125, 0.28662109375, 1.767578125, -1.4306640625, -1.0771484375, 0.7529296875, -2.349609375, 0.24267578125, -1.2255859375, 0.294189453125, -1.166015625, 1.2890625, -0.1441650390625, 1.2431640625, -0.04144287109375, 2.16796875, 1.810546875, -0.065673828125, 0.5009765625, -1.794921875, 0.27685546875, -0.286865234375, 1.1904296875, -0.494873046875, 1.01171875, 0.384521484375, 1.0224609375, 0.489013671875, 1.125, 0.3818359375, -0.0193023681640625, -0.287841796875, 0.5458984375, 0.22412109375, 0.91943359375, 0.0943603515625, -1.2822265625, -0.3564453125, 1.0126953125, -0.759765625, 0.1561279296875, -0.47216796875, -0.2259521484375, 0.1258544921875, -0.09503173828125, -0.058685302734375, 0.302734375, -1.5009765625, 0.6640625, 0.24072265625, -1.1982421875, -0.65380859375, 0.6787109375, -0.3955078125, -0.0845947265625, -1.5537109375, 0.4033203125, -0.61376953125, -0.0244293212890625, 1.091796875, 1.8212890625, 0.018096923828125, 0.5302734375, 0.304443359375, 0.521484375, -1.0771484375, -0.370361328125, 1.2109375, 0.3349609375, -0.322265625, -1.1337890625, 0.9287109375, -0.775390625, -2.03515625, -2.75, 1.2998046875, 0.9697265625, -0.029144287109375, -0.541015625, 0.619140625, 0.72802734375, -1.3525390625, -0.767578125, 0.341552734375, 0.5751953125, -0.4189453125, -0.92236328125, 0.057891845703125, 0.1422119140625, 2.130859375, -0.09405517578125, 1.02734375, 0.95654296875, -0.0285491943359375, 1.044921875, -1.5869140625, 0.556640625, -0.55126953125, 0.48779296875, 0.91796875, -1.03125, 0.234375, 0.481689453125, -1.6162109375, -0.059539794921875, 0.1531982421875, -0.269287109375, 0.257568359375, 0.56689453125, -0.06646728515625, 0.6103515625, -1.802734375, -3.54296875, 0.88671875, -0.0966796875, 0.446044921875, -0.194091796875, 0.0704345703125, -1.1962890625, -0.1390380859375, 0.79541015625, -0.0335693359375, -0.28662109375, -0.411376953125, 0.82421875, 1.1171875, 0.2174072265625, -1.029296875, 0.69140625, 0.4970703125, 0.26953125, -0.48193359375, 1.4384765625, 1.12109375, 0.77392578125, 0.767578125, 0.243896484375, 1.9404296875, -0.3349609375, -1.7685546875, -0.32861328125, -1.326171875, -1.7626953125, 0.2423095703125, -1.20703125, 1.1474609375, 0.339599609375, 0.333740234375, -2.171875, 2.048828125, 1.5888671875, -0.042694091796875, 1.361328125, -1.0361328125, 0.5986328125, -1.70703125, 0.347412109375, 0.8232421875, -0.72021484375, -0.258056640625, 0.984375, 1.4208984375, 1.3740234375, 1.3447265625, 1.1279296875, -1.3486328125, 0.484619140625, -0.8056640625, -1.0009765625, -2.10546875, -1.490234375, -0.213623046875, -0.0643310546875, 0.96337890625, -1.181640625, -0.6015625, 0.53564453125, -1.154296875, 1.1923828125, -1.3193359375, -1.3388671875, 2.208984375, -0.6044921875, 0.2454833984375, 0.6474609375, 0.305908203125, -1.31640625, -1.59375, -1.1376953125, -0.45458984375, -0.63818359375, -0.01215362548828125, 1.125, 0.525390625, -1.1181640625, 0.004852294921875, 1.3349609375, 0.98583984375, 1.43359375, 0.402587890625, 0.68359375, 2.357421875, 0.2822265625, -0.11614990234375, -1.38671875, 0.56494140625, 3.2890625, 0.59814453125, -1.8115234375, -0.38623046875, -1.6201171875, 1.025390625, 0.377685546875, -1.4306640625, -0.3486328125, 0.53955078125, 0.6083984375, -0.376220703125, -0.50927734375, 0.55908203125, -0.83349609375, -0.884765625, 0.7080078125, -0.78955078125, -0.300537109375, -1.310546875, 1.9775390625, 0.244140625, 0.50830078125, 0.86181640625, 0.43017578125, -0.1187744140625, 0.0845947265625, -0.177978515625, -1.2509765625, 0.3037109375, -0.417236328125, -2.142578125, 2.248046875, -1.16015625, 0.24755859375, -0.344482421875, -0.67919921875, -0.580078125, 0.67041015625, -0.5703125, -0.95849609375, 1.0078125, -0.61279296875, 0.07098388671875, 1.826171875, -0.455810546875, -0.00469970703125, -0.004962921142578125, 1.208984375, 1.7900390625, -0.32666015625, 0.705078125, 0.35498046875, -0.94189453125, -1.1884765625, 0.298095703125, -1.5625, 0.6494140625, -0.249267578125, 0.75048828125, -1.2919921875, 0.69189453125, -0.9248046875, 0.8125, -1.6787109375, -0.341796875, -0.39697265625, -0.276123046875, -0.52587890625, -0.349365234375, -1.8486328125, 1.9345703125, 0.296142578125, -0.7607421875, 0.6806640625, -0.5185546875, -0.564453125, 0.75634765625, 0.085693359375, 1.525390625, 0.8974609375, -1.66796875, 0.732421875, 0.888671875, -0.3193359375, -2.24609375, -0.193115234375, -1.55078125, 0.134765625, 0.299560546875, 0.056243896484375, 0.35498046875, 0.00911712646484375, -0.72216796875, 0.405517578125, 0.58544921875, 0.18017578125, -0.56884765625, 1.6171875, -0.884765625, 1.6123046875, -0.490966796875, -0.040130615234375, 1.3681640625, 0.01922607421875, -1.7978515625, -0.97900390625, 1.52734375, 0.7626953125, 1.7041015625, 0.7587890625, 0.1109619140625, -1.3232421875, 1.572265625, 1.0361328125, -0.484130859375, -0.33349609375, 0.66796875, -0.91162109375, 0.87158203125, 1.9169921875, -3.083984375, 0.615234375, -0.1712646484375, 1.0107421875, 1.0458984375, -2.037109375, -1.4169921875, -0.74658203125, -1.9072265625, 0.48681640625, 0.08135986328125, 0.283935546875, 0.62939453125, -1.3466796875, -0.229736328125, 1.607421875, 0.18701171875, -0.44873046875, -0.095947265625, -0.74609375, -0.1353759765625, 0.583984375, 0.45849609375, -0.07177734375, -1.298828125, -0.1822509765625, 0.34228515625, -1.2041015625, -0.44775390625, 0.61572265625, -0.9931640625, 1.384765625, -0.20068359375, 1.2705078125, -1.291015625, -0.384765625, 0.17822265625, -0.07470703125, -1.8212890625, -0.8037109375, 0.35986328125, 1.271484375, -0.5966796875, -0.331298828125, 0.79345703125, -0.23046875, -0.1832275390625, -1.189453125, 0.2022705078125, 0.68603515625, 1.2314453125, -0.12158203125, 0.40625, 1.513671875, 0.372802734375, -0.2705078125, -0.49267578125, -0.384765625, 0.050048828125, -1.6474609375, -0.09161376953125, -0.89990234375, -0.33642578125, 0.033050537109375, 0.290771484375, 0.537109375, -0.53466796875, 0.64306640625, 0.80810546875, -0.46923828125, -0.50390625, -0.410888671875, -1.0849609375, -0.40185546875, -0.0821533203125, -0.337158203125, 0.389404296875, 0.79443359375, -2.271484375, 0.8154296875, 0.94482421875, -2.39453125, 0.6455078125, 0.486328125, -1.1669921875, -0.272216796875, -0.2303466796875, 1.1748046875, -0.6279296875, -1.1650390625, 0.95166015625, -1.4560546875, -0.572265625, 0.84033203125, 0.51025390625, -0.55810546875, 0.173095703125, -0.201171875, -1.240234375, -0.40478515625, -0.84033203125, 1.275390625, -0.48388671875, 1.953125, -0.9384765625, 1.220703125, 0.97314453125, 0.1453857421875, 0.92138671875, -1.220703125, -1.2099609375, -1.107421875, 0.069580078125, 0.83056640625, -0.259033203125, -1.2578125, 0.060333251953125, -0.31298828125, -0.08721923828125, 0.52197265625, -0.6845703125, 0.09478759765625, 0.77685546875, 0.97216796875, -1.0029296875, 0.0743408203125, -0.78173828125, -0.08526611328125, -1.498046875, -0.37890625, -1.94140625, 1.146484375, 0.86572265625, 0.521484375, 0.96630859375, 1.140625, -0.3759765625, 1.3427734375, 1.71875, -0.379638671875, 1.3759765625, 0.2880859375, 1.154296875, -1.3740234375, 2.4921875, 0.875, -0.24951171875, 2.15234375, -0.040557861328125, 0.29931640625, 0.705078125, -0.50390625, -1.6083984375, 0.3642578125, 2.46484375, 1.6005859375, -1.294921875, 0.07843017578125, -0.9326171875, 0.67333984375, -0.023223876953125, -0.587890625, 0.58349609375, -0.8076171875, -1.025390625, -1.1318359375, 1.666015625, 2.146484375, 0.16796875, -0.455078125, -0.9228515625, 0.36669921875, 1.0087890625, 1.0966796875, 0.220703125, -1.6748046875, 0.53466796875, 0.07391357421875, -1.6064453125, -1.3369140625, -0.19677734375, -0.32568359375, 1.203125, 0.0673828125, -1.4345703125, -0.380859375, -0.75244140625, 0.2420654296875, -0.939453125, -2.689453125, -0.207763671875, 0.82568359375, 1.6484375, -0.8193359375, 0.1640625, 0.6728515625, 1.2138671875, 0.5908203125, 0.73095703125, -0.548828125, -1.4775390625, -0.297119140625, -0.69580078125, -0.86962890625, 0.1212158203125, 0.348388671875, 0.470703125, 0.1094970703125, 1.7822265625, -1.046875, -1.55078125, -1.134765625, -0.47802734375, 0.50732421875, -0.6201171875, -1.2236328125, 0.1531982421875, 1.6025390625, -1.8310546875, -0.67529296875, -1.505859375, 0.4931640625, -1.259765625, 1.0673828125, 0.459228515625, 0.1544189453125, 0.1705322265625, -1.255859375, -1.255859375, -1.1943359375, -1.1103515625, 0.09033203125, 0.316162109375, 0.487060546875, -0.83984375, -0.4765625, -1.1953125, -0.8916015625, -1.166015625, 0.12298583984375, -0.237060546875, 2.603515625, 0.82177734375, -1.9697265625, -1.2958984375, -1.0458984375, -0.2137451171875, 1.23046875, 2.390625, -1.484375, 0.420654296875, 1.1865234375, -1.94921875, 0.595703125, -0.2333984375, -0.6259765625, 1.791015625, 0.097412109375, 1.51953125, -0.0147247314453125, -0.953125, 0.4765625, -1.52734375, 2.3515625, -0.3447265625, 1.609375, 0.6240234375, -0.46484375, -0.8798828125, -0.460205078125, 0.71240234375, -1.16015625, -0.84912109375, 0.135986328125, -0.451904296875, 1.017578125, 1.0927734375, -0.72021484375, -0.6826171875, -0.5078125, 2.125, 0.58544921875, 0.6650390625, -0.7900390625, 0.53076171875, 0.76806640625, -0.505859375, 0.89794921875, -0.85107421875, 0.032867431640625, 1.1416015625, -1.625, 0.15869140625, 0.2174072265625, 1.4716796875, -0.0192108154296875, -0.374267578125, 0.55078125, 0.5810546875, 0.456787109375, 0.89892578125, -1.12890625, 0.197998046875, -0.08367919921875, 0.404052734375, 0.943359375, -0.1624755859375, -1.37890625, -1.7734375, -1.8955078125, -0.82958984375, 0.572265625, 0.455322265625, -0.393310546875, 0.163330078125, -2.189453125, 0.71533203125, 1.458984375, 0.650390625, -0.3583984375, 0.25390625, 1.21875, -0.160888671875, -1.642578125, -1.482421875, 0.8017578125, 0.3369140625, 0.6552734375, -0.31298828125, -1.294921875, -1.1220703125, -0.213134765625, 1.0546875, -1.33203125, -0.89599609375, -0.03790283203125, -0.8349609375, -0.04412841796875, 0.9541015625, -0.1429443359375, 0.423095703125, -0.374267578125, 0.80322265625, -0.73876953125, -0.32568359375, 0.552734375, 0.6318359375, 1.009765625, 0.55029296875, 1.5947265625, 0.373046875, -0.360595703125, 1.5107421875, -0.59033203125, 1.267578125, -0.2469482421875, 0.09857177734375, -1.4169921875, -1.1982421875, -0.136962890625, 0.79736328125, 0.59521484375, -1.7763671875, 0.259033203125, 0.322265625, -1.681640625, -0.52734375, 0.07464599609375, 0.96923828125, -1.568359375, -0.5146484375, -0.60107421875, -0.256591796875, -0.7919921875, -0.466064453125, -0.779296875, -0.1962890625, -1.6337890625, 0.90380859375, 0.424560546875, 0.068115234375, 0.452880859375, -0.223388671875, 2.515625, -0.64599609375, 0.249755859375, 0.25048828125, 0.12481689453125, -1.8671875, 1.5498046875, 0.54541015625, 0.64013671875, -1.390625, -1.5146484375, -0.79736328125, -0.468017578125, 0.069091796875, 0.2294921875, 0.06976318359375, -0.833984375, 0.370849609375, 2.013671875, -0.58935546875, 1.220703125, 1.4453125, 0.07586669921875, -0.179931640625, 0.841796875, 0.75830078125, -1.17578125, -1.9404296875, -0.39892578125, -0.75634765625, -0.031402587890625, -1.095703125, 0.16650390625, -0.061248779296875, 0.47705078125, 1.9765625, 1.3408203125, -0.44482421875, -0.19677734375, 0.1929931640625, 0.53662109375, -0.6328125, -1.119140625, 0.1859130859375, -0.29248046875, -0.80419921875, -0.72900390625, 1.4296875, 1.1064453125, -0.65771484375, 0.7333984375, 0.81689453125, -0.2098388671875, -1.0517578125, -1.98828125, -0.498291015625, -0.88720703125, -0.268798828125, -0.1275634765625, -0.59326171875, -1.38671875, 1.1162109375, 0.48779296875, -0.33837890625, -1.1396484375, -0.51904296875, -2.6796875, -0.27197265625, -0.9072265625, 1.6298828125, -0.03558349609375, 0.88330078125, -0.67431640625, -0.8720703125, -0.022979736328125, 1.380859375, 0.916015625, 0.1407470703125, -0.280517578125, -0.35107421875, 0.6865234375, -0.7509765625, -2.20703125, 0.41015625, 0.360107421875, 1.5537109375, -2.62890625, 1.10546875, 1.3544921875, 1.1875, -0.369140625, -0.62890625, -1.578125, -0.137939453125, 0.36279296875, 0.156494140625, 1.9619140625, 0.2177734375, -0.3212890625, -0.2626953125, -0.0156097412109375, -1.689453125, 0.09808349609375, -1.427734375, -1.029296875, 0.57568359375, -0.1571044921875, -0.85205078125, -1.5068359375, 0.3173828125, 0.91162109375, 1.1259765625, -0.06756591796875, 1.208984375, -0.36474609375, -0.60986328125, -0.1572265625, -0.06268310546875, 0.86572265625, 1.7138671875, -1.7724609375, -1.09375, -0.153076171875, -0.3447265625, 2.51171875, -0.57763671875, -0.6171875, -0.982421875, -0.677734375, -0.201904296875, 1.1845703125, 1.287109375, 1.697265625, -0.94189453125, 0.744140625, 0.9169921875, -0.58984375, 0.517578125, 0.775390625, 0.533203125, -1.1630859375, -0.57568359375, 0.458984375, -0.02606201171875, 2.021484375, 0.9423828125, 0.3779296875, -0.03118896484375, -0.060516357421875, -0.10430908203125, -0.64892578125, 0.358154296875, 0.1827392578125, 1.3408203125, 2.03515625, 1.3486328125, -0.318603515625, 0.269775390625, 0.422607421875, -0.50830078125, -0.324462890625, 0.76611328125, 0.5029296875, -0.18603515625, 0.081787109375, -0.78759765625, 1.2490234375, -0.241943359375, 0.058837890625, -0.492919921875, -1.158203125, 0.19140625, -0.042999267578125, -0.78076171875, 0.64404296875, -0.814453125, 0.513671875, 1.2685546875, 0.396728515625, -0.7607421875, 1.6533203125, 0.90185546875, 0.96142578125, -0.62353515625, 1.8466796875, -0.7666015625, 0.67919921875, -0.87451171875, -1.599609375, 1.76171875, -0.70654296875, -0.11480712890625, -0.027587890625, -0.47802734375, -1.296875, -0.81298828125, 0.2352294921875, 1.6669921875, -0.4033203125, 1.1611328125, -2.1328125, -0.55615234375, 0.771484375, -3.048828125, 0.449462890625, 0.17529296875, 0.5029296875, 0.35888671875, -0.29443359375, 0.5908203125, -1.0224609375, -2.412109375, 0.244384765625, 2.482421875, -0.54931640625, 0.521484375, 0.99560546875, -1.1572265625, -0.57080078125, 0.71337890625, -0.30419921875, 0.2509765625, -0.24853515625, -0.1103515625, -1.50390625, 0.1527099609375, 0.410400390625, 0.6962890625, -1.0205078125, 0.3017578125, 0.58251953125, -1.7529296875, -0.4013671875, 1.181640625, -1.5498046875, -0.58935546875, -0.6611328125, -0.58984375, 2.849609375, -0.876953125, 0.416748046875, 2.056640625, -0.471435546875, 0.060272216796875, -0.09320068359375, 1.5625, 0.830078125, 3.45703125, -0.405517578125, 1.046875, -0.30419921875, 0.355224609375, 0.41796875, 0.45068359375, 0.08612060546875, 0.2374267578125, -0.351318359375, -1.1689453125, -1.1591796875, -0.44775390625, -1.0556640625, -0.4384765625, -1.4501953125, 0.0634765625, 1.3828125, -1.30078125, 0.61083984375, 0.005542755126953125, -0.05584716796875, -0.74951171875, -0.9345703125, -0.30517578125, 1.41015625, -0.79150390625, -0.35302734375, -0.30517578125, -0.68994140625, 0.036376953125, -1.2431640625, -0.268798828125, 1.1962890625, -1.21484375, 0.658203125, -0.28955078125, -0.7421875, -0.79150390625, -0.016876220703125, 0.464599609375, -1.1357421875, 0.50244140625, 0.7080078125, -0.8310546875, 0.666015625, 1.150390625, -0.1859130859375, -0.287353515625, 0.43994140625, 0.75927734375, 1.107421875, 1.6005859375, -0.7763671875, -2.833984375, 0.282958984375, 0.61865234375, -0.31640625, -0.253173828125, 0.36962890625, -1.1533203125, -0.1658935546875, -0.3779296875, -1.4697265625, 0.77685546875, 0.57373046875, 0.72412109375, 0.0679931640625, 0.0241241455078125, -1.427734375, -1.1904296875, 0.053741455078125, -0.9306640625, 0.300048828125, 1.09765625, -0.99072265625, 2.181640625, 1.4052734375, 0.693359375, -0.283935546875, -0.8115234375, -0.389892578125, 0.716796875, -0.69189453125, 0.595703125, -1.232421875, -2.56640625, 0.80615234375, 0.1385498046875, -1.498046875, 0.66162109375, 0.921875, 1.3193359375, -2.0234375, 1.4306640625, -0.74853515625, 0.1900634765625, 2.390625, -0.681640625, -0.89501953125, -0.8720703125, 0.3984375, -0.97265625, -1.5087890625, -0.55712890625, 0.293212890625, -0.265869140625, 0.2020263671875, 0.6474609375, -0.345947265625, -0.349365234375, 0.70068359375, 0.0067291259765625, -0.29736328125, 0.28466796875, -1.4482421875, 1.5498046875, 0.033172607421875, 0.11578369140625, -0.72216796875, 0.1409912109375, 0.2191162109375, 0.1273193359375, -0.482421875, -0.5380859375, -2.880859375, -0.153076171875, -0.95703125, -0.7705078125, -0.1966552734375, 0.36083984375, 0.278076171875, 0.353515625, -0.017852783203125, -0.08843994140625, 0.99365234375, 1.0390625, 0.2242431640625, -0.399658203125, -1.455078125, 0.368408203125, 1.9677734375, -0.72900390625, -0.94873046875, 0.440185546875, -1.08203125, 0.888671875, 0.521484375, -0.182373046875, -1.1455078125, -0.131103515625, 1.068359375, -1.5068359375, 1.5595703125, -0.83056640625, 0.064453125, 0.87255859375, -0.384033203125, -1.4892578125, 0.140380859375, 0.415771484375, 1.2998046875, 0.9091796875, -0.111572265625, -1.4951171875, 0.5078125, 0.603515625, 2.599609375, -0.51708984375, 0.39306640625, 0.2293701171875, 0.225830078125, 0.50732421875, 2.70703125, -0.430419921875, -0.01251220703125, -1.064453125, 1.69140625, -0.429931640625, 0.4951171875, -0.59326171875, 0.2156982421875, 0.180908203125, 1.3583984375, -0.79931640625, 0.1536865234375, -1.26953125, 1.3486328125, 0.00901031494140625, -1.4638671875, -0.2083740234375, -0.68115234375, 0.849609375, 1.3125, -1.287109375, -0.919921875, -0.5556640625, 0.054473876953125, -0.3427734375, -2.021484375, 0.2220458984375, 0.75146484375, 0.3828125, 1.119140625, 0.81396484375, 0.70849609375, -0.51904296875, -0.0107879638671875, -1.2607421875, 0.439697265625, -0.501953125, 2.619140625, 0.001678466796875, -0.6396484375, -1.689453125, -1.41796875, 0.90234375, 2.640625, 0.05389404296875, 0.363525390625, 0.1650390625, -1.1640625, -1.4150390625, 0.53466796875, -2.3125, -0.86328125, 0.53955078125, -0.05291748046875, -0.47607421875, 0.7119140625, -0.284912109375, -0.456787109375, 0.830078125, -0.58251953125, 0.82421875, 0.0780029296875, -0.261474609375, 0.066162109375, 0.047698974609375, 0.5185546875, 1.1767578125, -1.4150390625, -1.255859375, -0.08428955078125, -0.77978515625, 1.0205078125, 1.4189453125, 0.86767578125, -0.2498779296875, 1.4599609375, -0.2171630859375, -0.822265625, 0.7431640625, 0.232421875, 0.51611328125, -1.2666015625, -1.08984375, -2.453125, 0.0665283203125, -0.40234375, -0.173095703125, 0.5908203125, -1.5361328125, -1.068359375, 0.734375, -0.0311431884765625, 0.9521484375, 2.17578125, -1.4111328125, 0.861328125, 0.08734130859375, 0.388916015625, 0.58056640625, -0.46923828125, 0.06671142578125, 0.38818359375, 1.5576171875, 0.55322265625, 1.0400390625, 1.068359375, -0.85009765625, 0.139404296875, 0.384765625, 0.76025390625, -1.2431640625, 0.81396484375, -0.53564453125, -0.0753173828125, 1.125, -1.0927734375, -0.2254638671875, -0.81591796875, 1.0791015625, -0.615234375, 1.1474609375, 0.90966796875, 0.8212890625, -1.349609375, -0.1964111328125, 0.0272674560546875, 0.458984375, 0.66748046875, 1.150390625, 0.496337890625, 1.251953125, 0.10931396484375, 0.179931640625, 1.3662109375, 1.1318359375, 0.3916015625, -0.463134765625, 1.48828125, 0.8583984375, 0.25830078125, -0.7275390625, 1.6796875, 1.4755859375, 0.79638671875, -0.6181640625, 1.0927734375, -1.56640625, 0.4111328125, -0.412353515625, 0.58251953125, -0.1583251953125, 0.64892578125, -1.38671875, 0.72607421875, 1.2783203125, 0.4638671875, 0.353759765625, -0.96923828125, 1.0087890625, -0.11834716796875, -0.1551513671875, -0.05255126953125, -0.459228515625, -0.755859375, -0.091552734375, 0.50927734375, 0.311767578125, 1.404296875, 1.908203125, -0.58935546875, -0.71240234375, -0.06671142578125, 0.3095703125, -1.654296875, -1.291015625, -1.1455078125, -1.6015625, -1.27734375, 0.72314453125, 0.8603515625, -1.77734375, 1.857421875, -0.0199737548828125, 0.92138671875, -0.137451171875, -1.0791015625, -1.4521484375, -0.046051025390625, -0.3740234375, 1.552734375, 2.099609375, -0.4287109375, -0.035980224609375, -0.4677734375, -0.0276641845703125, -0.01525115966796875, -0.91748046875, -2.3046875, -0.82470703125, -1.318359375, -1.24609375, -0.11712646484375, -0.0236053466796875, 1.3955078125, 0.734375, 1.7412109375, 0.43212890625, 1.533203125, 0.99462890625, 1.181640625, -0.73095703125, 0.316650390625, -0.2269287109375, -0.400390625, -0.7353515625, -0.83984375, 1.18359375, -0.5888671875, 1.1533203125, -1.021484375, 0.449462890625, -0.65625, 0.254150390625, -1.537109375, 1.8212890625, -0.59521484375, -0.5361328125, 0.1732177734375, 0.37109375, 0.7509765625, -0.177734375, 0.444091796875, 0.134521484375, -0.348876953125, 0.44287109375, 1.7490234375, 0.76123046875, 0.35791015625, -0.0104827880859375, -2.1875, -0.45166015625, -1.07421875, -0.66162109375, -0.09979248046875, 3.021484375, 0.1934814453125, -1.01953125, 0.2239990234375, 1.3466796875, 1.103515625, -0.82421875, -0.8017578125, 1.353515625, -0.7900390625, -0.28173828125, -0.1650390625, -0.685546875, 0.82470703125, 0.81005859375, 0.34716796875, -0.884765625, 1.1328125, 0.11199951171875, 0.45458984375, -0.41162109375, -2.529296875, -0.061004638671875, -1.033203125, 0.7158203125, 0.208984375, 0.046417236328125, 1.4873046875, -0.6962890625, -0.404296875, 0.57275390625, -0.76220703125, -0.76953125, 0.198974609375, -0.67724609375, -0.44580078125, 1.1142578125, 0.2841796875, -0.1522216796875, 1.0361328125, 0.77197265625, -0.5576171875, -2.078125, -1.01953125, 0.51318359375, -0.48291015625, 0.74267578125, 0.826171875, -0.11163330078125, 0.58642578125, -0.42529296875, 0.11376953125, -0.466552734375, 0.233642578125, -0.13720703125, -0.59423828125, -1.25, 0.24462890625, -0.37158203125, -0.223388671875, -0.43115234375, 1.38671875, -0.98828125, 0.888671875, 1.7626953125, 0.939453125, 0.06976318359375, 1.3330078125, -0.84423828125, -2.33203125, 0.93896484375, 0.51904296875, -1.7587890625, -2.38671875, 0.67138671875, 0.43017578125, 0.035430908203125, -0.529296875, 0.8544921875, -1.3916015625, 0.64208984375, -0.5244140625, 1.6181640625, -1.4384765625, -1.2197265625, 1.1357421875, -1.2001953125, -0.23388671875, 0.0184783935546875, 0.258056640625, -1.951171875, 2.123046875, -0.5546875, 1.19921875, 0.32275390625, -1.5419921875, 1.640625, -0.5224609375, 0.2158203125, 2.08203125, -1.0625, -0.9453125, -0.61865234375, 1.123046875, -1.1689453125, 1.2109375, 0.78271484375, 0.7421875, -1.3359375, 0.2352294921875, 1.0703125, 0.6669921875, 0.7001953125, 0.638671875, -1.482421875, -0.1484375, 0.1767578125, -0.26123046875, 0.783203125, -0.7763671875, -1.6142578125, 1.4169921875, -0.6826171875, -0.38720703125, 0.86865234375, 1.13671875, 1.0517578125, 0.004955291748046875, -0.351318359375, -0.4755859375, -0.70361328125, -0.99951171875, -0.537109375, 1.1650390625, -0.10009765625, -0.494140625, -0.88037109375, -2.380859375, 0.03631591796875, 0.216796875, -0.381103515625, 0.316650390625, 0.109619140625, -0.30615234375, -1.931640625, -0.11053466796875, -1.26171875, -0.78515625, -1.5107421875, 1.1025390625, -1.126953125, -0.279296875, 1.478515625, -0.85302734375, -0.0034332275390625, 2.0625, -1.619140625, -0.349609375, -1.2177734375, 1.9521484375, 0.72314453125, 0.66357421875, 0.1290283203125, 0.002811431884765625, 0.59375, 1.7021484375, -0.175537109375, 0.34619140625, -2.087890625, -0.56884765625, -0.025604248046875, 0.073486328125, -0.373291015625, 0.9580078125, 1.974609375, -0.5263671875, -0.03631591796875, 1.0830078125, -1.3935546875, -0.40576171875, -0.293701171875, -2.552734375, 0.12347412109375, 0.1563720703125, -2.28125, 1.1220703125, -0.8349609375, -0.94091796875, -0.7060546875, -1.10546875, -0.92138671875, 0.0633544921875, -0.286865234375, 0.275634765625, 0.337646484375, 0.424560546875, 0.87255859375, -1.203125, -0.51904296875, -1.193359375, -1.580078125, -0.4033203125, 2.380859375, 0.401123046875, 1.1318359375, -1.072265625, 1.5078125, 0.0406494140625, -1.841796875, -2.56640625, -0.962890625, 0.0791015625, 0.11181640625, -1.166015625, -0.0498046875, 0.521484375, -0.348388671875, -0.78271484375, 0.51708984375, 0.330810546875, -0.56103515625, 0.281494140625, -0.669921875, 1.89453125, 1.2763671875, 1.2373046875, -1.390625, -0.79833984375, 0.251708984375, -0.8857421875, 0.912109375, 1.5439453125, 1.7275390625, -0.349853515625, -0.09320068359375, 0.72314453125, 1.353515625, 0.06304931640625, -0.473388671875, -1.181640625, -0.53662109375, -0.69873046875, -1.181640625, -0.019989013671875, -0.2166748046875, 0.269287109375, 0.650390625, 0.6728515625, 0.2459716796875, -1.4169921875, 1.1181640625, -2.00390625, 0.479248046875, -0.10882568359375, 0.427734375, -0.034027099609375, 1.1435546875, 0.339111328125, -2.396484375, 1.19921875, 0.318115234375, -0.66796875, 0.38720703125, 0.7490234375, 0.62109375, -1.23046875, -0.340576171875, -0.28173828125, -0.92041015625, 1.638671875, -0.87109375, -1.7587890625, -1.2783203125, -0.546875, 1.279296875, -0.4072265625, -2.029296875, -0.8916015625, 1.576171875, 2.251953125, -0.359619140625, -0.65673828125, -1.5908203125, 0.7333984375, -1.4765625, -0.043914794921875, -0.2337646484375, 1.4140625, 1.880859375, 0.599609375, 1.869140625, -0.384033203125, 0.0966796875, -0.51904296875, -0.0977783203125, 0.681640625, -0.68212890625, 0.86767578125, -0.040802001953125, 0.302490234375, 1.5234375, -0.7138671875, 0.474609375, 0.07061767578125, 0.183349609375, -1.240234375, -0.402587890625, 0.31689453125, -1.5703125, -0.54541015625, -0.9228515625, -1.0185546875, 0.50732421875, 0.6220703125, -1.3447265625, 1.1220703125, -0.3955078125, -2.5703125, 1.04296875, -1.0107421875, 1.2724609375, 0.654296875, -0.546875, 0.1649169921875, 0.7109375, -0.046112060546875, -0.0357666015625, 1.044921875, 0.330322265625, 0.27001953125, -0.5234375, -0.74658203125, -0.54296875, -0.016632080078125, 0.221923828125, -0.68896484375, -0.51904296875, -0.7998046875, -0.187255859375, -1.5966796875, 0.026947021484375, -0.8837890625, 1.4482421875, -0.1788330078125, 0.1575927734375, 0.464599609375, 1.0068359375, 0.87255859375, -0.80078125, 0.308349609375, 0.1475830078125, 0.9580078125, 0.234619140625, 0.378173828125, 0.54541015625, 0.412353515625, -0.020050048828125, 0.134033203125, 1.580078125, -0.5322265625, 0.431396484375, 0.81201171875, -2.1484375, 1.244140625, 1.8955078125, -0.08428955078125, -0.83251953125, -0.017547607421875, 1.64453125, -1.509765625, 0.14306640625, -1.3857421875, 1.4638671875, -1.03125, -0.77587890625, 0.6513671875, -0.541015625, 0.0162200927734375, -0.59033203125, 1.6669921875, -2.19140625, -0.78759765625, -0.191162109375, -1.2294921875, -0.8447265625, 1.2939453125, 0.93017578125, 0.45947265625, 0.08416748046875, 0.54931640625, -0.5771484375, -1.765625, -0.81494140625, 0.1519775390625, -1.1923828125, 0.9091796875, 1.888671875, 0.1802978515625, 1.0478515625, -0.1492919921875, 0.004726409912109375, 1.5634765625, -0.4189453125, -0.31201171875, -0.10589599609375, -0.6142578125, -0.322998046875, -1.0947265625, 1.3134765625, 1.9404296875, 2.484375, -0.27978515625, 1.0830078125, 0.513671875, 1.3828125, 0.5068359375, -1.2080078125, -0.482666015625, -1.671875, 2.068359375, 0.77490234375, 1.2900390625, 1.5859375, 1.2666015625, -0.87890625, 0.1966552734375, -1.4892578125, 0.751953125, -0.1715087890625, -0.1214599609375, 0.50927734375, -0.61474609375, 1.5234375, 0.72998046875, -1.89453125, 0.0269622802734375, 0.69287109375, -0.440673828125, 0.474365234375, 0.41015625, 0.64501953125, 0.71728515625, -1.134765625, -0.45751953125, -0.68603515625, 1.0908203125, -0.480224609375, -0.04083251953125, 0.07061767578125, -0.83056640625, -1.2978515625, 0.2098388671875, 0.59423828125, -0.00479888916015625, -0.9384765625, 0.6142578125, 0.2318115234375, 1.28515625, -0.77587890625, -2.150390625, 0.708984375, -0.8115234375, -0.462890625, -0.71435546875, -0.1483154296875, -0.67529296875, -0.06768798828125, -0.76611328125, 0.445068359375, -0.99267578125, -0.75732421875, 0.7568359375, -0.154052734375, 0.86572265625, -0.5703125, -1.2763671875, -1.12890625, -0.0694580078125, -0.46435546875, -2.40234375, 0.51318359375, 1.2451171875, 0.32666015625, -2.068359375, -1.072265625, 1.3369140625, 0.69775390625, 0.5234375, -0.1497802734375, -0.73583984375, 1.7373046875, 0.5322265625, -2.70703125, 1.23046875, 0.10418701171875, -0.047882080078125, -0.96435546875, 1.9443359375, -2.4296875, 1.2802734375, 0.783203125, 1.1787109375, -0.393798828125, 0.6640625, 0.341796875, -1.1953125, 0.368408203125, 0.10223388671875, -0.52783203125, 0.39013671875, -0.184326171875, 1.3017578125, -0.7568359375, 2.275390625, -1.9677734375, -1.337890625, 0.67822265625, 0.307373046875, 0.1044921875, 1.3291015625, 1.6171875, 1.0908203125, -0.4375, 0.473876953125, -0.370849609375, 0.68505859375, -0.70556640625, -1.3779296875, -0.810546875, 0.2310791015625, 0.92724609375, 0.8916015625, 1.48828125, 0.2421875, 0.35302734375, -0.5078125, -2.75, -0.51220703125, -0.63720703125, -0.326171875, 0.58447265625, -0.51025390625, -0.3720703125, 1.30078125, 0.2161865234375, 0.927734375, -0.050018310546875, 0.1580810546875, -1.6259765625, 0.87646484375, -1.7578125, -0.63037109375, -1.9013671875, -0.8974609375, -1.37109375, -1.9912109375, -0.96875, 1.82421875, -1.009765625, 0.83203125, -0.1995849609375, -0.066162109375, 0.94775390625, 0.1546630859375, -0.11602783203125, -0.1357421875, 2.09375, -0.09423828125, -0.338134765625, 1.2412109375, -0.19580078125, -0.41455078125, -0.43310546875, 0.1888427734375, -0.1502685546875, -0.274169921875, -0.822265625, -0.91943359375, -1.0556640625, -0.68603515625, -1.0361328125, 0.51171875, -1.935546875, -0.36474609375, -0.7431640625, 0.10870361328125, -1.3310546875, -0.5810546875, -0.2333984375, -0.469970703125, -0.765625, 1.625, -0.050567626953125, 0.61376953125, 2.4609375, 0.671875, -1.09375, 0.244140625, 1.2353515625, 1.712890625, -0.5654296875, 0.60888671875, -0.58935546875, 0.87451171875, -0.37353515625, -0.68212890625, -0.1429443359375, -0.80712890625, -1.279296875, 1.3125, -0.353759765625, 3.28515625, 1.3642578125, 2.138671875, -0.376953125, 0.623046875, 0.0232086181640625, 1.873046875, 1.41015625, 0.9541015625, -1.4609375, 0.826171875, -0.4404296875, 1.0654296875, -0.2607421875, -1.1533203125, -0.2998046875, -0.43310546875, 0.14306640625, -0.1810302734375, 0.90380859375, -0.59130859375, 0.58935546875, -0.186279296875, 1.3525390625, -0.1287841796875, -1.0634765625, -0.658203125, 1.076171875, 1.205078125, -0.428466796875, -0.2469482421875, -1.455078125, 0.79248046875, 0.53564453125, -0.580078125, 1.44140625, -1.205078125, 0.377685546875, 1.23828125, -0.1787109375, -0.413330078125, 0.87353515625, -0.373779296875, 0.12481689453125, -0.2763671875, 0.5732421875, -1.1494140625, 0.98046875, 1.2060546875, 0.2379150390625, 0.64501953125, 0.77001953125, 1.1005859375, -0.378173828125, -0.06512451171875, 0.408447265625, -0.31640625, 1.525390625, -1.017578125, 1.015625, 0.465576171875, 1.1025390625, 1.7783203125, -0.282470703125, 0.548828125, 0.80224609375, 0.0345458984375, -0.85009765625, -0.226806640625, 0.1888427734375, 1.5107421875, 2.234375, -0.89892578125, 1.24609375, 0.12445068359375, 0.1456298828125, -0.3232421875, -0.034210205078125, 1.3193359375, 0.460205078125, -0.88037109375, 0.467529296875, 0.8076171875, -0.025299072265625, -0.2027587890625, -0.187744140625, 0.93603515625, -1.3974609375, -0.02105712890625, 0.382080078125, 1.07421875, 1.0390625, 1.0068359375, -1.3017578125, 0.39453125, 0.57568359375, 0.27392578125, -0.36474609375, -0.60498046875, -0.7763671875, 1.1328125, -0.1556396484375, -0.630859375, -0.9609375, -0.70947265625, 2.041015625, 0.06402587890625, -2.451171875, -0.407470703125, 1.4375, -0.9296875, 0.5673828125, -1.2666015625, -0.85546875, -1.6767578125, 0.44287109375, 1.1875, 0.888671875, 0.26806640625, 1.6181640625, 0.11749267578125, 0.17919921875, -1.38671875, -0.77294921875, 0.482177734375, 1.38671875, -2.099609375, 1.748046875, 2.009765625, 0.47509765625, -0.61962890625, -0.313232421875, 0.75244140625, -0.31884765625, -0.3291015625, 1.833984375, -0.08306884765625, -0.64990234375, -0.026092529296875, -0.221923828125, 0.20947265625, 0.1988525390625, 1.7783203125, 0.5263671875, -0.98388671875, -0.2420654296875, -0.386962890625, -1.0068359375, 0.035430908203125, -1.078125, 0.73828125, 0.5361328125, -0.25732421875, 1.197265625, 0.434814453125, -2.08984375, -1.443359375, 0.8466796875, 0.4697265625, -1.5263671875, -0.78076171875, 1.12109375, -2.20703125, 0.82421875, -0.417236328125, 1.353515625, 1.8486328125, 1.2001953125, -0.552734375, 0.0117034912109375, 1.2021484375, -1.4423828125, 0.039276123046875, -1.21484375, -2.052734375, 0.2210693359375, 0.146728515625, 0.12548828125, 1.0576171875, -0.71826171875, -0.873046875, -0.6357421875, -0.93896484375, -0.8564453125, 0.7841796875, 0.11944580078125, 0.77099609375, -0.8681640625, -1.04296875, 1.47265625, 0.218017578125, 0.58154296875, -0.062744140625, -0.34814453125, -1.087890625, -0.483642578125, -0.30517578125, 0.21484375, -0.42724609375, 0.2296142578125, -0.59375, -0.57568359375, -0.7314453125, -0.2509765625, 1.521484375, 1.2744140625, 0.81689453125, -0.225341796875, -0.62548828125, -1.4072265625, -0.476318359375, 0.74072265625, -1.2109375, -0.58935546875, 1.80859375, -0.334716796875, 1.1962890625, 0.2454833984375, -0.1697998046875, 1.0458984375, -0.1002197265625, 0.87548828125, 0.41064453125, -2.171875, 0.0093536376953125, -1.5224609375, 0.55615234375, -0.724609375, 2.068359375, -0.859375, -0.332763671875, -0.384765625, 0.434326171875, -0.2357177734375, 1.8388671875, -2.052734375, -1.0537109375, -1.01171875, -1.052734375, -0.763671875, -0.89306640625, 1.5869140625, 1.072265625, -0.4814453125, 0.453369140625, 0.8740234375, 1.2177734375, 1.49609375, 2.3828125, -0.642578125, 0.31884765625, 0.036773681640625, -0.054534912109375, 0.472900390625, -0.65966796875, -1.626953125, 0.171630859375, -0.028045654296875, -0.222900390625, 1.9296875, 0.515625, 0.361083984375, 0.88818359375, -1.017578125, 2.30859375, 1.6171875, -0.64453125, -0.203125, 1.0859375, -0.36474609375, -0.11968994140625, -0.6826171875, 0.11767578125, 2.646484375, 1.0810546875, 0.8798828125, -1.515625, -0.05828857421875, -0.59765625, 0.6748046875, -0.908203125, -0.94189453125, 0.52490234375, -1.4990234375, 0.1815185546875, -0.15087890625, 0.521484375, 0.67529296875, -1.2216796875, 0.310546875, 0.08074951171875, -1.0439453125, 0.302001953125, -0.521484375, 0.5576171875, -0.7822265625, 0.9140625, -0.5107421875, 1.517578125, 0.220458984375, 1.015625, -0.61962890625, 1.3173828125, 0.99365234375, 0.463623046875, -0.259521484375, -0.5615234375, 0.3408203125, -0.2315673828125, -0.1873779296875, -1.251953125, 0.1400146484375, -0.55615234375, 0.97314453125, 1.236328125, 1.25390625, -0.66357421875, -1.08203125, -1.2158203125, -0.352783203125, -1.212890625, 0.828125, -0.0374755859375, -0.8662109375, 0.08990478515625, -0.1851806640625, 1.076171875, -0.25927734375, -0.578125, -0.142822265625, 0.53564453125, -0.75634765625, 0.12481689453125, -1.1865234375, -1.1142578125, 1.529296875, 1.82421875, 0.63720703125, 0.6201171875, -1.6484375, 0.241943359375, -0.45751953125, -2.28125, -0.225830078125, 1.0869140625, -1.37109375, 0.546875, 0.404052734375, 0.12469482421875, -0.218505859375, -1.044921875, -0.59814453125, -0.537109375, 1.1728515625, -1.6328125, -0.5673828125, -0.912109375, 0.002132415771484375, -0.92236328125, -0.77587890625, -0.157470703125, 1.3857421875, 1.0302734375, 0.260986328125, 0.68994140625, 0.7734375, 0.089599609375, 1.392578125, 1.60546875, 0.97509765625, -0.560546875, -0.258544921875, -0.703125, -0.326171875, -0.4599609375, -1.3056640625, -0.73486328125, 0.2164306640625, 0.63720703125, 0.8916015625, -1.0126953125, 0.48095703125, -0.4677734375, -0.65185546875, -1.021484375, 0.1832275390625, -0.5771484375, -1.017578125, 0.59619140625, 0.299072265625, -0.66796875, -0.990234375, -0.74462890625, -0.11767578125, 0.29736328125, -0.51318359375, -1.158203125, -1.82421875, 0.6064453125, -0.75830078125, -0.06048583984375, 0.1356201171875, -0.382568359375, 1.919921875, -0.92529296875, -1.2177734375, 0.36767578125, -0.515625, -0.41259765625, 0.0137481689453125, 0.88671875, 1.005859375, -0.03851318359375, 0.455078125, 2.10546875, 1.2060546875, 0.5869140625, -0.91455078125, -0.1357421875, -1.349609375, -0.1534423828125, 0.08282470703125, 1.8818359375, 0.176025390625, 0.156494140625, 1.3662109375, -1.2939453125, 1.4541015625, 0.5927734375, 1.318359375, -0.74267578125, -1.048828125, -0.61474609375, 0.07049560546875, -0.80029296875, -0.79931640625, 0.036468505859375, 0.650390625, 0.8310546875, -0.60986328125, 1.4970703125, 0.69775390625, -0.6240234375, -1.0390625, -0.1849365234375, 0.29638671875, -1.123046875, -0.0223846435546875, -2.16015625, -1.013671875, -0.94091796875, 0.284423828125, 1.1103515625, 1.76171875, 0.01531982421875, 0.98388671875, -2.521484375, -0.47412109375, 0.9296875, 0.359375, 0.5322265625, 0.1268310546875, 1.7587890625, 1.349609375, -0.459716796875, -0.7685546875, 0.28955078125, 0.01212310791015625, -0.53564453125, -0.07080078125, 0.56005859375, 1.828125, 0.0108184814453125, -1.029296875, -0.1197509765625, -0.3818359375, 0.247802734375, -1.078125, 0.2452392578125, -2.529296875, 1.1533203125, 0.71728515625, -0.138671875, -2.978515625, 1.3515625, 0.87060546875, -2.9140625, -1.4375, 0.92431640625, 0.437744140625, -0.61865234375, 0.865234375, -0.07965087890625, -0.427734375, 0.9384765625, -0.6787109375, -1.3251953125, -0.62744140625, 2.1484375, -0.8251953125, 1.11328125, 0.389892578125, -2.515625, -1.208984375, 0.90234375, -0.578125, 1.4052734375, 3.24609375, -1.7822265625, -0.4892578125, -0.188232421875, 0.94091796875, 0.0947265625, -0.431396484375, -0.69873046875, -1.587890625, -0.460205078125, 0.451416015625, -0.1610107421875, 0.576171875, -1.0302734375, -0.025421142578125, 0.1171875, -0.1434326171875, -1.0751953125, 0.9814453125, -0.78662109375, 0.381103515625, -1.7900390625, 0.466064453125, 1.2158203125, -1.5361328125, 0.6943359375, -2.287109375, -0.180419921875, -1.109375, 1.1435546875, -2.0546875, 0.432861328125, -0.2763671875, -0.66015625, -0.9775390625, -1.1201171875, 1.5810546875, 0.347412109375, 1.3095703125, 0.414794921875, 1.25, -0.304443359375, 1.2265625, 0.293212890625, 0.032867431640625, 0.3564453125, -1.7509765625, -1.7197265625, -2.09375, -0.3154296875, -0.78955078125, 2.515625, 0.2568359375, 0.54833984375, -0.448974609375, 0.384521484375, -0.93701171875, 2.02734375, -1.767578125, -0.31396484375, -0.7158203125, 0.9208984375, 1.3232421875, 1.22265625, 0.04180908203125, -1.96875, 0.04766845703125, -1.09765625, -0.218017578125, -1.01953125, -0.267578125, 0.2347412109375, -0.302001953125, 0.2181396484375, -1.0517578125, -0.0797119140625, 0.16259765625, 1.16796875, 0.81884765625, 1.41015625, 1.876953125, -1.4931640625, -0.87841796875, -1.19140625, -0.283935546875, 0.45703125, -0.52587890625, -1.052734375, 1.0888671875, 0.57470703125, 0.0819091796875, 1.6279296875, -0.259521484375, -0.373046875, -0.369873046875, 1.466796875, 0.2481689453125, -0.236572265625, 0.56201171875, 0.2646484375, -0.46240234375, -0.59033203125, -1.5087890625, 0.342041015625, -1.0546875, 1.087890625, -0.06072998046875, 0.241943359375, 0.89404296875, 0.162353515625, -1.1025390625, 0.5693359375, -1.1181640625, -0.130126953125, -0.27392578125, -0.49365234375, 1.1650390625, 0.27294921875, 0.49462890625, 3.21484375, 1.33203125, 0.59375, -0.6142578125, -0.0985107421875, 0.303955078125, -0.3671875, 0.8837890625, 1.181640625, 0.666015625, 0.578125, 0.41455078125, -0.84765625, 0.022613525390625, -0.900390625, -0.5673828125, -1.3447265625, -1.5703125, 1.0341796875, -1.69140625, 0.638671875, -0.00931549072265625, 0.38671875, 1.341796875, 0.29541015625, -0.497314453125, -0.94482421875, 1.2685546875, -0.15869140625, -1.5048828125, 0.296875, 0.0889892578125, 0.2314453125, -1.2060546875, -0.6005859375, -0.302001953125, -0.54931640625, 0.3046875, -0.260498046875, 0.59423828125, 1.0517578125, -0.61181640625, -0.392822265625, 0.1492919921875, -0.11181640625, 0.79736328125, -1.318359375, 0.8115234375, -1.4228515625, -2.0625, -1.90625, -0.84912109375, -1.0625, -1.4326171875, -1.2236328125, -0.61767578125, -0.282470703125, 0.279052734375, 0.200439453125, 0.53955078125, -2.34375, 1.93359375, -1.291015625, 1.697265625, 1.0244140625, 2.173828125, -0.036956787109375, -0.6669921875, 0.83544921875, -0.455078125, 0.27490234375, -0.08355712890625, -0.86328125, 0.68603515625, -0.53125, 1.20703125, -0.56298828125, -0.7109375, -1.7568359375, -0.09075927734375, -0.986328125, -0.0193634033203125, 1.1201171875, 0.17333984375, -0.39013671875, 0.89306640625, 1.216796875, 0.3935546875, -0.62060546875, 0.9970703125, -1.474609375, 0.1219482421875, -0.59033203125, 1.837890625, -0.68798828125, -0.79638671875, 0.6767578125, 0.673828125, 0.873046875, 2.55859375, 0.5048828125, -0.195556640625, 0.140625, -0.9501953125, 0.62451171875, 0.188720703125, 1.568359375, -0.23291015625, -0.6982421875, 0.31689453125, -0.1947021484375, 1.7431640625, -0.9287109375, 1.03515625, -1.2333984375, -0.626953125, 0.17529296875, -0.407958984375, -0.564453125, 0.4462890625, -0.24169921875, 2.201171875, 0.875, -1.5380859375, 1.1396484375, 0.179931640625, 0.0589599609375, 2.2890625, 0.65478515625, 0.42724609375, -0.05926513671875, 0.06884765625, -0.95556640625, -0.55908203125, 0.0165252685546875, -0.2369384765625, 0.0816650390625, -0.06011962890625, 0.7744140625, 2.048828125, -0.021514892578125, 0.58544921875, 0.69482421875, -1.537109375, 0.076416015625, -0.0953369140625, -0.57568359375, 0.55615234375, 1.51171875, -0.75341796875, -0.0121307373046875, -0.07391357421875, 0.412109375, -0.6943359375, 1.5419921875, 0.384765625, 1.60546875, -0.84521484375, 0.08843994140625, -1.2548828125, 1.3037109375, -0.71826171875, -1.0673828125, -0.8681640625, 1.9677734375, 1.0107421875, 1.28125, -0.98388671875, -0.95703125, 0.28076171875, 0.0222015380859375, 0.95556640625, -0.76513671875, -1.8173828125, -1.6865234375, 0.370849609375, -0.60302734375, -0.50439453125, 0.2012939453125, -0.2269287109375, -2.533203125, -0.8095703125, 2.14453125, -0.9580078125, 0.53076171875, 0.479248046875, -1.7490234375, 1.4072265625, -0.98583984375, -1.619140625, -0.89453125, -0.294677734375, -0.16796875, 0.208984375, 0.59228515625, 0.2445068359375, -0.65673828125, -0.681640625, -0.5869140625, -0.0506591796875, -0.2071533203125, 0.271240234375, -0.318603515625, -0.2412109375, 0.035888671875, 0.5087890625, 1.552734375, -0.175048828125, 1.4755859375, 0.26904296875, -0.81396484375, 0.1737060546875, 1.560546875, -0.6220703125, 1.009765625, -0.7314453125, 0.258056640625, 0.57080078125, 1.740234375, -0.381591796875, 0.53564453125, -0.4228515625, -0.1405029296875, -0.47021484375, -0.467041015625, 1.2890625, -0.73583984375, 0.36474609375, -1.7373046875, -0.6318359375, 2.052734375, -1.4951171875, 0.2010498046875, 0.0738525390625, 0.439208984375, 0.005001068115234375, 2.00390625, -0.39013671875, 0.411376953125, -0.086181640625, 0.431884765625, 0.06964111328125, -1.12890625, -0.34228515625, 0.67529296875, 1.1962890625, -0.81396484375, 0.654296875, -0.1690673828125, 0.6572265625, -1.509765625, 0.7421875, -0.7216796875, 2.224609375, -0.2423095703125, -0.06854248046875, -0.79833984375, -1.337890625, 0.994140625, 0.10723876953125, 1.5712890625, -0.3828125, 1.0712890625, -0.467529296875, 1.8857421875, 0.09002685546875, -0.54541015625, 1.728515625, -0.25244140625, 0.6162109375, 0.339599609375, -1.7607421875, 0.2216796875, -2.15625, -0.89306640625, -1.119140625, -1.337890625, 1.7412109375, -0.1572265625, 0.69189453125, -0.26806640625, 1.0634765625, 1.1064453125, -0.345703125, -0.268798828125, -1.166015625, 0.90087890625, -0.1392822265625, 2.287109375, -1.4013671875, 1.021484375, -0.654296875, 0.6962890625, -0.98193359375, 0.2666015625, 0.429931640625, -0.1907958984375, 1.2197265625, 1.162109375, -0.61083984375, -2.212890625, 1.2138671875, -0.28955078125, -0.46630859375, -0.44140625, -0.258544921875, 0.95068359375, -1.6142578125, -0.1962890625, -0.9375, 0.192138671875, 1.3798828125, 0.7265625, -0.1583251953125, 0.062255859375, 1.1455078125, 1.1708984375, -0.06494140625, -1.5322265625, -0.298095703125, -0.40087890625, -0.2548828125, 0.345458984375, 2.015625, -0.061309814453125, 0.244140625, 0.75537109375, -0.79248046875, -0.6416015625, -1.072265625, 0.19580078125, 1.677734375, -0.9384765625, 0.2210693359375, -0.427978515625, 0.1427001953125, 0.8408203125, -0.8486328125, 1.21875, 0.83056640625, 0.274658203125, 0.045135498046875, -0.833984375, 0.88134765625, -0.974609375, 1.6884765625, -1.3125, 2.466796875, 1.716796875, 0.69580078125, -1.30078125, 1.6240234375, -1.705078125, -0.3740234375, -0.201416015625, -0.919921875, -0.22265625, 0.220947265625, -0.69140625, 0.364013671875, -0.065185546875, 0.286376953125, -0.1597900390625, -1.8095703125, -0.94091796875, -0.26513671875, -0.6181640625, -0.52099609375, -0.057830810546875, -1.7607421875, -1.7900390625, -0.297119140625, 0.525390625, 0.021728515625, 0.0034275054931640625, 0.3193359375, -1.3603515625, 0.1624755859375, 0.37744140625, -1.5712890625, 0.330078125, -0.8876953125, -0.17138671875, -1.2890625, -0.85498046875, 0.071044921875, 0.78173828125, 0.56494140625, 0.07305908203125, -0.01007843017578125, -1.1171875, 0.03314208984375, -0.4755859375, 1.2236328125, 0.28662109375, -0.290771484375, -0.74609375, 1.482421875, -0.10870361328125, -1.4482421875, -0.185546875, -0.385498046875, -0.2071533203125, -0.442626953125, 0.477294921875, 0.02825927734375, 1.3056640625, -1.171875, -0.1724853515625, -0.94921875, 0.67431640625, 0.5810546875, -0.1654052734375, -1.1162109375, 0.275146484375, -0.2607421875, -1.1494140625, -0.09722900390625, 0.81201171875, -1.36328125, 2.1796875, 0.2271728515625, -0.371337890625, -0.341796875, 0.3515625, -0.27099609375, -0.435791015625, -1.328125, -0.63330078125, 0.24169921875, 0.60693359375, 0.9580078125, 1.578125, 0.75732421875, 1.4482421875, -0.364990234375, -0.465087890625, 0.156005859375, 1.28125, 0.212646484375, 0.1820068359375, -1.1025390625, 2.162109375, 0.024444580078125, -0.39453125, 0.83740234375, 0.146484375, -1.5107421875, -0.054779052734375, -0.02435302734375, 0.92578125, -2.037109375, 1.4267578125, -1.208984375, 0.65771484375, -0.1348876953125, 0.45068359375, 0.199462890625, 0.3359375, -0.30908203125, -0.3193359375, -0.229248046875, -2.140625, -0.002368927001953125, 0.423095703125, 1.455078125, -0.40283203125, -0.2281494140625, 0.83740234375, 0.61962890625, -0.15087890625, -0.177978515625, -1.302734375, 0.157958984375, -0.373779296875, -0.97119140625, 0.0489501953125, -0.76171875, -0.34814453125, -1.720703125, 1.1220703125, -1.296875, -0.50439453125, -0.1329345703125, -0.2264404296875, 0.5869140625, -0.07244873046875, 1.59375, 0.12274169921875, 1.697265625, 0.73193359375, -0.1502685546875, 0.53955078125, 0.64599609375, 1.1923828125, 0.428466796875, -0.8173828125, 1.2568359375, -0.9443359375, 0.318115234375, 0.038238525390625, 1.939453125, 1.9140625, 0.2115478515625, 1.072265625, 0.393798828125, -1.2109375, -1.517578125, 0.499267578125, 1.84375, -0.11334228515625, -0.634765625, -1.0400390625, -0.0258636474609375, -0.54296875, -0.21826171875, 0.1937255859375, -2.283203125, 0.609375, 1.0048828125, 0.33349609375, 1.4189453125, 0.89453125, 1.138671875, -2.771484375, -0.08990478515625, 0.67529296875, 0.356689453125, 0.134033203125, -0.70947265625, -0.5185546875, -0.039093017578125, -1.5205078125, 0.285400390625, 1.8427734375, -1.0107421875, 0.51220703125, -0.8935546875, 1.4814453125, 0.9072265625, 0.419189453125, 1.2158203125, 1.8515625, 1.7353515625, 1.109375, 0.21923828125, -0.650390625, 1.109375, -1.75390625, -2.9765625, -0.3349609375, 0.155029296875, -0.36328125, 0.329833984375, 2.41015625, -0.042144775390625, -0.26806640625, 0.68994140625, -1.2744140625, 1.8896484375, 0.046875, 0.31201171875, -0.4404296875, 0.3525390625, -0.55029296875, 0.1422119140625, 0.251220703125, -0.31396484375, -0.54296875, 0.52587890625, 0.630859375, -0.7529296875, 1.7353515625, -0.90625, 0.346923828125, -0.187255859375, 0.2171630859375, 1.169921875, -0.338623046875, 1.662109375, -2.205078125, -0.94384765625, -0.465576171875, -1.0029296875, 1.6201171875, 1.580078125, 0.177490234375, -1.3994140625, -0.505859375, 1.724609375, -0.376220703125, 2.62109375, 0.82275390625, 0.3701171875, -0.54052734375, -0.354736328125, -0.004055023193359375, 1.623046875, 1.2236328125, 0.8203125, -0.419189453125, -1.775390625, -1.0703125, -0.97705078125, 0.2103271484375, -0.41015625, -0.10101318359375, -1.3193359375, 0.20751953125, -1.4375, -1.7978515625, 0.35791015625, 0.90576171875, -0.1373291015625, 1.716796875, -0.11138916015625, 0.021514892578125, -0.42578125, 0.253662109375, 0.71142578125, 0.1826171875, 0.321533203125, -2.75390625, -0.798828125, 0.01036834716796875, -0.6328125, -0.209228515625, -1.3798828125, 1.650390625, 1.4140625, 0.72998046875, 0.0216827392578125, 1.291015625, 0.9345703125, 1.263671875, 0.7529296875, 0.33447265625, 0.423828125, 1.0595703125, 3.39453125, -1.2607421875, 1.013671875, -0.0157318115234375, 0.1405029296875, -0.64501953125, 1.7861328125, 1.2109375, -1.7216796875, 0.46044921875, -0.6279296875, -1.607421875, -0.383544921875, -0.267578125, 1.5009765625, -0.0447998046875, 1.3583984375, -0.1763916015625, -0.425048828125, -0.71630859375, 1.4052734375, -0.0782470703125, 1.060546875, -1.2607421875, 0.2225341796875, 0.243408203125, -0.58984375, -0.94384765625, 0.84814453125, 0.34912109375, 0.455322265625, 1.6474609375, -0.95458984375, 0.8681640625, 0.75927734375, 0.057220458984375, -1.7900390625, -0.00919342041015625, -0.9326171875, 0.89990234375, -0.327880859375, -0.2724609375, 1.435546875, -0.292724609375, 0.8779296875, -0.12152099609375, -0.1815185546875, -0.4296875, 0.453857421875, 0.351806640625, -1.357421875, 0.609375, 0.276611328125, -0.87255859375, -0.07940673828125, 0.29150390625, -1.158203125, -1.0400390625, -0.1875, 0.7265625, 3.09765625, 0.8203125, -1.619140625, 0.1717529296875, -1.1845703125, 1.744140625, 0.4208984375, -0.44384765625, 0.443359375, -1.68359375, 0.52392578125, -0.345947265625, 0.736328125, 0.59326171875, 1.1630859375, 1.7568359375, 1.4755859375, 0.82275390625, -0.263916015625, -0.1539306640625, -0.35595703125, 0.404541015625, 1.0234375, -0.72265625, -0.1419677734375, 0.5390625, 0.67919921875, -0.032135009765625, -0.06121826171875, 1.505859375, 0.2071533203125, 0.7548828125, -0.11541748046875, -0.5771484375, 0.62744140625, 0.9482421875, 1.4619140625, -1.2197265625, -0.0250091552734375, 0.39453125, 0.243896484375, -1.4990234375, 0.92333984375, 0.83203125, 0.306640625, 1.373046875, 1.126953125, 0.6962890625, 1.2900390625, -0.1920166015625, 0.9970703125, -0.09759521484375, -0.95654296875, 1.4619140625, 0.646484375, 0.02349853515625, 1.66796875, 0.62353515625, -0.705078125, 1.0517578125, 0.27783203125, -0.06939697265625, 0.9560546875, 0.369384765625, 0.89111328125, -0.455810546875, -1.49609375, -0.46533203125, 0.408203125, 1.6513671875, 1.5791015625, -1.1884765625, 0.5478515625, 0.2039794921875, -1.0986328125, -0.139892578125, -1.2255859375, -0.0109405517578125, -0.070068359375, 0.0362548828125, -1.4365234375, 0.07855224609375, 0.52783203125, -1.0732421875, 0.75341796875, 1.2294921875, -0.417236328125, -0.69482421875, 0.45654296875, 0.5625, 0.483154296875, 0.5927734375, -0.131591796875, -0.0499267578125, -1.5263671875, -0.53369140625, -0.317626953125, -0.163330078125, 1.115234375, 1.9248046875, -0.8076171875, 0.411865234375, -0.34814453125, -1.1376953125, -0.479736328125, 0.2232666015625, 1.0322265625, 0.7158203125, 0.59130859375, 0.1944580078125, 0.272216796875, -2.2265625, -1.1796875, 0.6259765625, 0.5400390625, -0.96142578125, 1.2568359375, 0.5849609375, -0.2216796875, 1.06640625, 0.52392578125, 1.4140625, 0.39892578125, -0.88330078125, 0.498046875, -0.397216796875, 0.54833984375, 0.32861328125, -2.09765625, -2.275390625, 1.0263671875, 0.249755859375, 1.6005859375, 0.306396484375, 0.234130859375, 1.138671875, -0.429443359375, -1.2099609375, -0.63623046875, 0.006595611572265625, 0.234619140625, 0.81591796875, 1.0654296875, -0.20263671875, -1.6240234375, 1.294921875, -0.8955078125, 0.2890625, 0.10491943359375, 0.128662109375, -0.329833984375, 0.55712890625, -1.3359375, -0.59228515625, -0.2083740234375, -2.052734375, -0.4658203125, -0.77197265625, -1.453125, 0.97119140625, 0.07708740234375, 0.1524658203125, 0.67236328125, -1.6171875, -0.037872314453125, 1.73828125, 0.896484375, 0.064453125, -0.6923828125, 0.333984375, 1.9453125, -1.0732421875, 0.81591796875, 0.509765625, 0.806640625, -0.390869140625, -0.9423828125, 1.0634765625, 1.09765625, -0.8466796875, -0.46728515625, -0.30615234375, -1.66796875, -0.30859375, -0.1611328125, 1.041015625, -0.52099609375, -1.3310546875, -3.10546875, 0.470947265625, -0.74560546875, -0.046142578125, -1.09765625, 1.2451171875, 2.296875, -0.1861572265625, -0.347900390625, 1.03515625, 0.1180419921875, 1.875, -1.625, -1.1572265625, -0.1273193359375, -2.140625, -0.1968994140625, 0.68701171875, -0.9599609375, -1.21875, -0.29638671875, -1.5146484375, 1.31640625, 1.076171875, 0.775390625, -0.0703125, 0.509765625, -0.33642578125, -0.41552734375, 0.5263671875, 1.2568359375, -0.1810302734375, 0.354736328125, 0.95947265625, 2.146484375, 2.361328125, 0.39404296875, -1.455078125, -0.69970703125, 1.3154296875, -0.53076171875, -1.775390625, 2.013671875, -0.34716796875, -0.86328125, -1.2119140625, -0.1884765625, -0.5830078125, 1.6240234375, -1.1416015625, -0.0677490234375, -1.8095703125, 0.01287078857421875, -0.6806640625, -0.2469482421875, -0.2252197265625, 0.5615234375, 0.3720703125, -0.51708984375, 0.787109375, -0.1636962890625, 1.1591796875, 0.70947265625, 1.2001953125, -0.90869140625, -0.1829833984375, 0.76513671875, 0.34912109375, 0.1728515625, 0.33837890625, 0.0823974609375, -1.1943359375, -0.070068359375, 0.152587890625, 0.363037109375, 0.60205078125, -0.962890625, -0.29248046875, -0.41064453125, -0.08697509765625, -0.55615234375, 0.31396484375, -0.0657958984375, -0.53759765625, -1.892578125, -0.5517578125, -1.5126953125, -0.037994384765625, 0.2484130859375, -0.1365966796875, -1.037109375, -1.1181640625, -1.4072265625, -1.1103515625, 0.896484375, -0.4951171875, -1.044921875, 0.52294921875, 0.54541015625, 0.86767578125, 0.245849609375, -1.0634765625, 1.173828125, -0.55712890625, -0.0266876220703125, 0.432373046875, 0.47314453125, -0.269287109375, -0.33837890625, -1.0087890625, 0.720703125, -1.521484375, 0.4580078125, 1.0966796875, -2.09765625, -0.094970703125, 0.27685546875, -0.96337890625, 0.7333984375, -0.389404296875, 2.0859375, 0.10357666015625, -1.51953125, -1.5849609375, -1.1435546875, -0.226806640625, -1.115234375, -0.533203125, -1.6826171875, -2.35546875, 1.169921875, 1.49609375, 1.767578125, 1.32421875, 1.0703125, 0.63916015625, 1.4716796875, -0.55224609375, 0.55908203125, -1.5205078125, 1.36328125, -0.96044921875, -0.09747314453125, 1.7490234375, 1.0263671875, 0.85400390625, -0.10040283203125, 0.583984375, 1.0849609375, 1.017578125, 0.74853515625, 0.48388671875, 0.04937744140625, 0.62744140625, 0.75341796875, -0.82373046875, 1.4521484375, 0.098876953125, 0.1590576171875, 0.45263671875, 0.2783203125, -0.658203125, -0.01181793212890625, -0.658203125, -0.75390625, 1.85546875, 1.263671875, -0.425537109375, -1.2236328125, 0.02239990234375, 1.6142578125, 0.055511474609375, -0.7685546875, -1.0361328125, -0.6728515625, 2.064453125, 1.5068359375, -1.3212890625, -0.16552734375, -0.11663818359375, -0.470703125, 0.284423828125, 1.400390625, 0.069091796875, 0.75634765625, -0.353759765625, 0.212646484375, -0.9501953125, 0.33056640625, -0.034698486328125, 0.65673828125, 0.288330078125, 1.9892578125, 1.095703125, 0.42041015625, -0.42724609375, -1.3271484375, 0.395751953125, 0.8271484375, -0.517578125, 0.51318359375, 1.5185546875, 0.498291015625, 0.05914306640625, 0.9892578125, -1.228515625, 0.3291015625, -2.71484375, -0.4345703125, -0.8994140625, -0.80029296875, -0.5859375, -0.767578125, 1.2421875, 0.59375, 0.4619140625, 0.74609375, 1.236328125, -0.2158203125, 1.251953125, 0.8017578125, -1.3671875, -0.9853515625, -1.7021484375, -0.966796875, 1.568359375, 0.031982421875, -1.0537109375, -0.1331787109375, -0.517578125, 1.3291015625, 1.201171875, -0.98828125, -0.46435546875, -0.8291015625, 0.69921875, -1.0029296875, -1.4326171875, 0.0462646484375, 1.7216796875, 0.01226806640625, -1.2109375, -0.19189453125, -1.02734375, 0.396728515625, -0.75146484375, -1.2333984375, -0.1341552734375, -1.162109375, -0.4140625, -0.80419921875, -0.51611328125, 1.4267578125, -0.583984375, -0.307373046875, 0.06036376953125, 0.9296875, -0.697265625, 0.246337890625, 0.95849609375, -0.3017578125, -1.0146484375, 0.2330322265625, 1.3623046875, 1.1044921875, -0.546875, -0.02276611328125, 0.0007891654968261719, -0.16015625, -0.334716796875, -0.2335205078125, 0.3515625, 0.1278076171875, 0.73388671875, -0.440673828125, -0.1708984375, 0.321044921875, 0.5009765625, -2.505859375, -0.83251953125, 1.7861328125, 0.388671875, -1.02734375, 0.62890625, 0.44580078125, 2.037109375, -0.97265625, -0.0186767578125, 0.681640625, -0.338623046875, 0.806640625, 1.48828125, 0.01033782958984375, 0.3642578125, 0.68896484375, -0.3056640625, -1.2099609375, 1.0498046875, 0.60791015625, 0.2978515625, 0.78076171875, -0.1783447265625, -0.60400390625, -1.3291015625, -0.80517578125, 1.56640625, 1.0947265625, -0.58203125, -1.34765625, -0.287109375, 0.84765625, -1.5927734375, 0.418701171875, -0.1776123046875, -2.236328125, 0.260498046875, 0.59130859375, 0.2174072265625, 1.072265625, 0.153564453125, -1.146484375, 0.196044921875, -0.68994140625, -0.10101318359375, 1.8173828125, 0.61181640625, -0.196044921875, 2.314453125, -0.0180511474609375, -0.978515625, 1.14453125, -0.01444244384765625, 0.416748046875, 0.444091796875, -0.87841796875, -1.0361328125, -1.2919921875, 1.2490234375, -0.4677734375, -0.98828125, -2.607421875, -0.0211029052734375, 1.2666015625, -2.072265625, -0.8515625, 0.26220703125, 1.111328125, -0.29833984375, 0.78857421875, 0.0264129638671875, 0.84765625, 0.61962890625, -0.016998291015625, 0.73486328125, -0.0222320556640625, -1.345703125, -0.32666015625, -0.301025390625, 0.86083984375, 0.438720703125, 0.75146484375, 1.69921875, 1.37109375, 0.2467041015625, 0.058074951171875, 0.97802734375, -0.2841796875, -0.6259765625, -0.5302734375, -0.2880859375, -0.216552734375, 1.794921875, -0.39892578125, 0.09539794921875, 1.0009765625, 0.53466796875, -0.4765625, -0.73583984375, 0.08612060546875, 0.6962890625, -0.341552734375, 1.1650390625, 0.3369140625, -0.315673828125, 0.197021484375, -0.70654296875, -0.7392578125, 1.017578125, -0.42138671875, -0.5029296875, 0.46826171875, -0.1661376953125, 0.4423828125, -0.32080078125, -1.1337890625, -0.1767578125, 0.2430419921875, -0.03546142578125, -1.357421875, -0.265869140625, 0.71875, -1.31640625, -0.728515625, 0.7138671875, 0.38916015625, 0.046661376953125, 0.5322265625, 0.919921875, -0.87548828125, 1.7841796875, 0.8994140625, 0.5205078125, -2.203125, 0.0281829833984375, -1.1123046875, -0.8388671875, 0.2208251953125, -1.400390625, -0.2437744140625, -2.068359375, 1.1357421875, 0.272705078125, -0.07208251953125, -0.8759765625, -1.2578125, 0.4423828125, 0.2200927734375, -0.09674072265625, 0.771484375, -0.28466796875, -1.73828125, 0.08221435546875, 0.6083984375, 0.56982421875, -0.900390625, -0.81884765625, -1.1064453125, 0.040252685546875, -0.7939453125, 0.5966796875, -0.39404296875, 1.064453125, 0.7763671875, 0.5849609375, 0.39892578125, -0.0164642333984375, 0.5869140625, -1.291015625, 0.05169677734375, 0.2371826171875, 0.349609375, -1.947265625, -0.4853515625, 0.391357421875, 1.4921875, -0.39501953125, 1.0087890625, -0.0511474609375, 0.58984375, -0.728515625, -0.1021728515625, 0.0831298828125, -2.609375, 1.0849609375, 1.001953125, -0.288818359375, -0.05169677734375, 0.10302734375, 0.97216796875, 0.66748046875, 0.88232421875, 1.3408203125, 0.2216796875, 0.161376953125, -0.87744140625, -2.576171875, 0.294677734375, -0.5927734375, 2.453125, 0.12481689453125, -0.273681640625, -0.97607421875, -0.055206298828125, 0.309326171875, -1.333984375, -0.495849609375, 1.26171875, 1.8095703125, -0.0743408203125, 0.37353515625, -0.455322265625, 0.1654052734375, -0.044952392578125, 0.07025146484375, 0.480224609375, 0.060211181640625, 2.345703125, -0.81689453125, 1.4482421875, 1.10546875, 0.587890625, 0.08795166015625, -0.11834716796875, 2.294921875, -0.03802490234375, -1.6357421875, -0.0687255859375, 0.304443359375, -0.365478515625, 2.71875, 0.333740234375, 0.71142578125, 0.8193359375, 0.69970703125, -0.02734375, 0.1978759765625, -0.46044921875, -0.7138671875, -0.26611328125, 0.609375, 0.64697265625, 0.6748046875, -0.97802734375, 0.7451171875, -0.0869140625, -1.4716796875, 0.5751953125, 0.6748046875, -0.93115234375, 1.7900390625, 0.005893707275390625, 0.23583984375, -0.20703125, 0.33154296875, 0.2030029296875, -0.420654296875, -0.59765625, -0.458740234375, -2.61328125, -0.2314453125, 0.43701171875, -0.873046875, -0.52197265625, -0.60107421875, -1.6806640625, 0.373291015625, -0.96142578125, -0.423095703125, -0.76220703125, -0.404541015625, 1.091796875, 0.1590576171875, 0.452392578125, -1.314453125, 0.491943359375, 0.87841796875, 1.158203125, -0.026214599609375, -0.794921875, 0.93115234375, -0.6064453125, -1.943359375, 0.1978759765625, -0.83544921875, 1.1337890625, -0.406494140625, -0.1712646484375, -0.156982421875, 0.1275634765625, 0.087646484375, 2.859375, -1.01953125, 0.5966796875, -1.044921875, 0.122314453125, -0.3076171875, 1.1982421875, -0.98046875, 0.7666015625, 1.8916015625, -0.489990234375, 1.54296875, 1.123046875, 0.35205078125, -0.425048828125, 0.62890625, -2.42578125, -0.89697265625, 0.71337890625, 0.040618896484375, -1.16015625, -0.66845703125, 0.3203125, -0.63037109375, -0.9892578125, -1.5615234375, 0.333984375, -0.5498046875, 1.5380859375, 1.0849609375, 1.005859375, 2.12890625, 0.203369140625, -0.239501953125, -0.030975341796875, 0.1943359375, 0.9716796875, -0.3486328125, -0.767578125, -1.28125, -1.6630859375, -0.0008530616760253906, -0.2646484375, 0.086669921875, 0.748046875, -1.240234375, -0.0028533935546875, -0.1468505859375, -0.089111328125, -0.384765625, -1.0224609375, 0.73388671875, -2.0546875, 0.1278076171875, 1.2880859375, -0.392578125, -0.5888671875, -0.79931640625, 1.0791015625, -0.12109375, -0.377685546875, -0.266357421875, -0.99365234375, 0.8076171875, 1.1240234375, -1.625, -1.9404296875, 0.195556640625, -0.266357421875, 0.7880859375, 0.4208984375, 0.927734375, -0.440185546875, 0.77490234375, 2.75, -0.7998046875, -0.23291015625, 0.077392578125, 0.705078125, -0.74560546875, -0.475341796875, -0.59375, 0.25537109375, -0.56689453125, -0.74658203125, 0.2841796875, 0.261474609375, -0.1787109375, 0.42431640625, 0.44140625, -0.1165771484375, -0.35693359375, -0.92138671875, 0.267822265625, -0.626953125, 1.8017578125, 0.2305908203125, 1.2236328125, 0.77978515625, -0.57421875, -0.5048828125, -1.14453125, -0.92041015625, 0.336669921875, 0.2451171875, -0.705078125, 0.420166015625, -0.15625, 0.321044921875, -0.70947265625, 1.00390625, -0.68212890625, -0.072509765625, 1.0, 1.0537109375, -0.6640625, -1.01171875, 0.62841796875, -0.79296875, -0.732421875, -1.603515625, 1.173828125, 0.41845703125, 0.60791015625, 0.1787109375, 0.47705078125, 1.1904296875, -1.7080078125, -0.638671875, -0.2413330078125, 0.427734375, -1.3720703125, -0.43408203125, 1.2978515625, -0.0394287109375, 0.17529296875, 0.041717529296875, 0.1248779296875, 0.04681396484375, -0.51416015625, 0.480224609375, -0.4775390625, -0.5341796875, 0.2291259765625, -1.3271484375, -0.232421875, 0.0226287841796875, 0.55712890625, 0.5, -1.8515625, 0.32421875, -0.8427734375, -1.0986328125, -1.693359375, -1.1591796875, 0.779296875, 0.271484375, 0.177978515625, 0.55029296875, 0.148681640625, 0.03131103515625, 0.94921875, -0.51806640625, 0.06640625, 0.0002925395965576172, -0.927734375, 1.421875, -1.1904296875, -0.28369140625, -0.88037109375, -3.369140625, 1.572265625, 1.01171875, -1.24609375, -1.189453125, -1.1572265625, 1.03515625, -0.58740234375, 0.201171875, 0.1912841796875, 0.021209716796875, 1.0703125, -0.10858154296875, -1.392578125, -1.6064453125, 0.327880859375, 2.201171875, 2.16015625, -0.468994140625, 0.826171875, 0.70458984375, 1.197265625, 1.0634765625, -0.55029296875, -0.859375, -0.7177734375, 0.32763671875, 0.4619140625, 0.4013671875, -0.07989501953125, 0.73779296875, -1.5419921875, 3.36328125, 0.1553955078125, 0.09844970703125, 0.377197265625, 0.6875, 0.05523681640625, 0.69091796875, 0.4970703125, 0.10711669921875, -2.2109375, -0.459228515625, -1.185546875, -1.0126953125, -0.677734375, -1.0849609375, -1.01953125, 1.5048828125, 0.95654296875, -0.260498046875, 1.099609375, 0.204345703125, -0.51611328125, -0.55908203125, 0.03192138671875, 0.978515625, -0.8095703125, -0.87060546875, -0.537109375, 0.2880859375, 0.486328125, 0.48583984375, 0.383056640625, 0.369140625, -1.8330078125, -0.9208984375, -0.268798828125, -1.5595703125, -1.376953125, -0.305419921875, 0.45263671875, -2.181640625, 0.5810546875, -1.7685546875, -2.4140625, 0.1566162109375, 1.2294921875, -1.4658203125, 0.1737060546875, 0.1470947265625, 0.377197265625, 0.7138671875, 1.7734375, -1.3916015625, -1.390625, -0.043701171875, 1.1796875, 0.95947265625, -1.7705078125, -1.2861328125, 0.79931640625, -0.708984375, 0.426513671875, -1.404296875, 0.0908203125, 0.40869140625, -0.38916015625, 0.046234130859375, -0.473876953125, -0.313232421875, -1.2119140625, -0.4853515625, 0.0467529296875, 0.73681640625, -0.30078125, 0.8974609375, -0.26611328125, 1.302734375, -0.3134765625, -1.177734375, 0.0714111328125, -0.89892578125, -0.8212890625, -1.8095703125, -0.2032470703125, 1.3369140625, 1.8857421875, -1.4931640625, 1.4306640625, 0.75146484375, 0.42431640625, -0.26220703125, -0.43408203125, 0.2198486328125, 2.46484375, -0.86376953125, -1.173828125, 0.003513336181640625, 0.161865234375, 0.45654296875, -0.646484375, 0.93896484375, -0.287109375, 1.6787109375, -1.1044921875, -1.0283203125, 0.030670166015625, -0.77587890625, -0.7939453125, 0.52978515625, 0.9931640625, -0.152587890625, -0.468505859375, 0.277587890625, -1.36328125, -0.434326171875, 0.8876953125, -2.96875, 0.9609375, 0.83203125, -0.373291015625, -1.4326171875, 1.3369140625, 0.438720703125, 1.1591796875, 1.8662109375, -0.28515625, -1.0107421875, -0.95703125, 1.92578125, 1.01171875, 0.8544921875, 0.927734375, -0.4736328125, 0.38134765625, -0.6875, -0.95263671875, 0.53759765625, 0.9501953125, -0.1474609375, 0.62353515625, -0.31494140625, 0.477294921875, -0.83349609375, 0.275634765625, 1.1025390625, -0.133544921875, -0.81884765625, 1.234375, 1.623046875, -1.986328125, -0.2314453125, 0.8349609375, -0.451904296875, 0.300537109375, 1.69921875, 0.62158203125, -0.2169189453125, -1.2763671875, -1.3232421875, -1.248046875, -1.583984375, 0.0584716796875, -0.1353759765625, 0.9453125, -1.44921875, -0.77880859375, 1.388671875, 0.576171875, -0.01323699951171875, 0.57177734375, 1.4189453125, 1.1484375, 0.126220703125, -0.63720703125, -0.1385498046875, -1.427734375, -1.5205078125, -1.6591796875, -0.7998046875, -0.7822265625, -0.87646484375, 0.2470703125, 0.1912841796875, -0.654296875, 0.68896484375, -0.8603515625, 0.75244140625, 0.277587890625, -1.265625, 0.890625, -1.0126953125, -0.315673828125, 0.7841796875, 0.056427001953125, 0.7841796875, 2.294921875, 0.1590576171875, 0.085205078125, 1.7333984375, 0.455810546875, 0.138916015625, -0.0034637451171875, 1.2216796875, 1.5888671875, 0.351318359375, -0.01264190673828125, 0.39013671875, 0.6767578125, -0.54345703125, -0.459228515625, -0.374755859375, 0.35498046875, -0.74560546875, -0.671875, 1.625, 0.236572265625, 0.6533203125, -0.1278076171875, -0.732421875, 0.7578125, -1.1337890625, 0.81787109375, -1.1806640625, -1.8173828125, 0.88623046875, 0.576171875, 1.072265625, -0.46142578125, 1.75390625, 0.118896484375, -0.68994140625, 0.8681640625, 0.68701171875, -0.211181640625, -0.1292724609375, -1.7978515625, 0.73583984375, -1.6533203125, 0.88671875, 0.47021484375, -1.3759765625, -0.49072265625, 0.1658935546875, 1.16796875, 1.5439453125, 0.73583984375, 1.5791015625, -0.03125, 1.736328125, -0.765625, 0.98193359375, -0.10687255859375, -0.859375, -0.23681640625, 0.880859375, 0.329345703125, -0.1341552734375, -1.56640625, -0.043548583984375, -0.52099609375, 0.55126953125, 0.050201416015625, 0.51611328125, -0.81787109375, 0.53662109375, -0.06927490234375, 0.591796875, 0.33203125, -0.272216796875, -0.155029296875, 1.9033203125, 0.1654052734375, -0.1776123046875, -0.44873046875, 0.083984375, -0.164306640625, 1.0380859375, -0.26611328125, 0.31201171875, 0.31494140625, 0.130859375, -2.0703125, 0.031768798828125, 0.322021484375, -0.439208984375, 0.80712890625, 1.1923828125, 1.6416015625, -1.2626953125, 0.74609375, -1.9560546875, -1.548828125, 0.320556640625, 1.220703125, -1.6279296875, 0.59326171875, 0.5732421875, 2.212890625, 0.49609375, -0.6337890625, 1.1064453125, -0.7138671875, 1.294921875, -1.0888671875, 0.65625, 0.68798828125, -0.51904296875, 1.9921875, 0.028961181640625, 1.6259765625, -1.0712890625, 1.3271484375, -0.36767578125, -1.806640625, 1.08984375, -2.021484375, -1.119140625, 0.51708984375, -0.603515625, -0.09173583984375, -0.5615234375, -0.296142578125, -1.84375, -0.826171875, 0.489990234375, -1.31640625, -0.0821533203125, -0.890625, -0.2459716796875, 0.8017578125, 1.4716796875, 1.4794921875, 0.050567626953125, 1.0302734375, -1.2763671875, 2.408203125, 1.34375, 0.732421875, 1.279296875, -2.11328125, 0.20654296875, -0.3271484375, 0.83642578125, -1.8916015625, 0.2410888671875, -0.59716796875, -0.666015625, -2.1171875, -0.45068359375, -0.446533203125, 1.1337890625, -0.751953125, -0.79931640625, -0.80517578125, 1.857421875, -0.484130859375, -0.325927734375, 0.92578125, 0.362548828125, 0.91162109375, 0.634765625, -0.023895263671875, -0.544921875, -0.65625, -0.1651611328125, 0.06756591796875, 0.72802734375, 0.274169921875, 0.302001953125, -0.57373046875, -0.474609375, 0.392333984375, -0.47216796875, 0.31640625, -0.51171875, 2.1640625, 0.833984375, -2.248046875, -0.9873046875, 0.19287109375, -0.469482421875, -1.4111328125, -1.5283203125, 1.1689453125, -0.3916015625, -1.017578125, -1.4423828125, -0.8134765625, 0.65869140625, -0.186279296875, 2.73046875, -0.8291015625, 0.673828125, -0.382568359375, 0.422607421875, 0.80126953125, 2.734375, 0.51904296875, 0.07965087890625, 0.336669921875, -0.495361328125, 0.142822265625, -0.765625, 0.81787109375, -1.9677734375, -0.70068359375, 0.383544921875, -0.62646484375, -0.4521484375, 0.034271240234375, -0.34423828125, 1.3388671875, -0.376953125, 0.60009765625, -0.26708984375, -0.76904296875, 0.384521484375, 0.37841796875, -2.31640625, 0.44873046875, -0.84033203125, 1.607421875, -0.185546875, -0.048309326171875, 0.39697265625, -0.65087890625, -1.7265625, 0.460205078125, -0.39599609375, -0.8505859375, 0.84130859375, 0.9599609375, -0.2041015625, -0.402099609375, 0.11212158203125, 0.2159423828125, 0.8974609375, 0.7666015625, 0.93603515625, 0.1270751953125, -0.06890869140625, 1.923828125, -0.155517578125, -0.06304931640625, 1.220703125, -0.7919921875, -0.3349609375, 0.8271484375, 0.9248046875, -0.90087890625, -0.74658203125, -1.1875, 0.08050537109375, 0.392333984375, -2.34765625, 0.677734375, -0.73486328125, 0.78466796875, 0.7529296875, -0.1124267578125, -0.492919921875, -1.2060546875, -0.70654296875, 0.408203125, -0.134521484375, 0.630859375, -1.1787109375, 0.0236663818359375, -0.058563232421875, 0.57861328125, -1.486328125, 0.58349609375, 0.2489013671875, -0.449462890625, 0.1656494140625, 1.46484375, 1.623046875, -0.1871337890625, -1.61328125, -0.2205810546875, -0.43408203125, -0.75732421875, -6.133317947387695e-05, -0.4658203125, -1.15625, -0.6171875, -0.9990234375, 0.509765625, -1.3486328125, 0.10491943359375, -0.1580810546875, -0.3447265625, -0.61328125, -0.45947265625, 1.955078125, 0.8544921875, 0.005767822265625, -0.7841796875, -0.8388671875, 0.923828125, 0.8193359375, -0.462158203125, -0.043182373046875, -1.3427734375, -1.548828125, -0.142578125, -0.0050506591796875, -0.76025390625, -1.8994140625, 2.50390625, -0.599609375, 2.525390625, 1.333984375, 0.09820556640625, 0.2425537109375, 1.044921875, 1.4921875, -0.7587890625, -0.85107421875, 0.416259765625, -0.43603515625, 1.1806640625, 0.63134765625, -1.103515625, -0.5185546875, -0.60888671875, -1.3037109375, -0.0246734619140625, 1.77734375, 0.320556640625, -0.92724609375, -0.60009765625, -1.3369140625, -1.2783203125, 0.16552734375, 0.78662109375, 2.76953125, -0.54052734375, 1.8232421875, 0.34814453125, 1.7060546875, -0.07318115234375, -0.736328125, 0.80322265625, 0.15087890625, 1.2578125, 0.5390625, 0.35791015625, -0.5498046875, 1.228515625, -0.46044921875, 2.45703125, 1.8076171875, 0.3896484375, 0.169189453125, 0.03106689453125, 0.78662109375, 0.5888671875, -1.4384765625, -0.3388671875, 1.064453125, -0.381591796875, -0.052490234375, 0.20068359375, 1.1416015625, -1.154296875, 0.8330078125, 0.21923828125, -1.310546875, -0.6865234375, -0.46826171875, -1.5849609375, 0.57373046875, 2.234375, 1.1240234375, -0.60546875, 1.6171875, 0.1796875, 0.079833984375, 2.318359375, -0.50048828125, 1.3720703125, -0.56787109375, -0.043975830078125, 0.277587890625, -0.91162109375, 1.1337890625, -1.2431640625, 1.779296875, 0.8232421875, -1.703125, 0.6455078125, 0.10760498046875, -1.1826171875, 1.439453125, -0.0165557861328125, -0.497314453125, -2.0234375, -0.92724609375, -1.1474609375, 0.477294921875, -0.33544921875, -0.45263671875, -0.3759765625, 1.4541015625, 0.1490478515625, 1.580078125, -0.705078125, 1.08984375, 1.5341796875, 0.457275390625, 0.379150390625, 0.85791015625, -0.99755859375, -0.91259765625, 0.97314453125, 0.70458984375, 0.0601806640625, 0.2005615234375, 0.05096435546875, -0.08660888671875, -0.197998046875, 0.75732421875, 0.6884765625, -1.0576171875, 0.82470703125, -0.65234375, -0.515625, -1.2529296875, -1.640625, 1.5078125, -1.46875, 0.01097869873046875, 0.75732421875, 0.126953125, 0.53125, 0.90576171875, 1.14453125, -1.6572265625, 1.712890625, -0.01044464111328125, 0.298095703125, -1.537109375, 1.05859375, 1.3125, 0.70263671875, 0.04815673828125, -0.6259765625, -0.560546875, -0.8798828125, 0.43896484375, 2.533203125, 0.7109375, -0.1673583984375, 0.09674072265625, -0.8896484375, -0.44091796875, 1.6875, -1.1884765625, -1.546875, 0.95703125, 0.3994140625, -0.0016613006591796875, -1.4677734375, -1.31640625, 1.30078125, 1.39453125, 0.837890625, -0.74658203125, 1.1357421875, -2.369140625, -1.00390625, 1.74609375, -0.349609375, -2.11328125, -0.146728515625, -1.2470703125, 0.296142578125, -1.267578125, 1.611328125, 0.042266845703125, -0.2264404296875, -0.44921875, 1.3984375, -2.03125, -0.1893310546875, -0.5546875, -0.277099609375, -1.509765625, 1.3876953125, 0.361328125, -1.783203125, -0.059600830078125, -0.9072265625, 0.3984375, 0.410400390625, -0.11346435546875, -1.833984375, 0.307373046875, 0.44384765625, 0.252197265625, 2.17578125, -1.2998046875, -0.49951171875, -1.251953125, -2.0546875, 1.8095703125, 0.76513671875, -0.5751953125, 0.482421875, -1.115234375, 0.88525390625, 0.533203125, -0.66162109375, -0.047607421875, -0.431396484375, 0.296875, -0.322021484375, 1.78125, 1.470703125, 2.771484375, -0.66796875, 0.156494140625, 0.4345703125, -1.2353515625, -1.037109375, 0.2474365234375, 0.052520751953125, -0.8623046875, -1.8291015625, -0.9267578125, -0.42626953125, -0.482177734375, 0.58544921875, 0.943359375, 0.97119140625, 0.62158203125, 0.327880859375, -0.46484375, 1.1552734375, 0.70068359375, 0.6884765625, 1.1787109375, -0.17626953125, -0.56103515625, 1.0986328125, -0.2724609375, 0.5283203125, -0.2142333984375, 1.0400390625, -0.58544921875, -0.376708984375, 1.5439453125, -0.93603515625, -0.646484375, -1.60546875, 0.5458984375, 0.6083984375, -1.818359375, 0.927734375, 0.49609375, -0.474609375, 0.8779296875, -0.75244140625, -0.424560546875, 1.2529296875, 1.0625, -0.6943359375, -0.72314453125, 0.63037109375, -1.4619140625, -0.463623046875, -1.3193359375, -1.126953125, 0.6044921875, 0.83544921875, -0.2357177734375, 0.73486328125, 0.1888427734375, -1.5283203125, 0.111572265625, -1.130859375, 0.97412109375, 0.316162109375, -1.1728515625, -0.26171875, -1.310546875, -0.55322265625, 1.0263671875, 0.0171356201171875, -0.13916015625, -1.0888671875, 0.356201171875, -0.21484375, -0.69970703125, 1.9208984375, 1.060546875, -0.875, 1.0888671875, 0.4912109375, -0.1036376953125, 0.2164306640625, 1.1201171875, -0.001308441162109375, -1.4375, -0.7783203125, 1.1708984375, -0.74755859375, 0.26416015625, -0.8935546875, -0.1300048828125, -0.1845703125, 0.8828125, -0.288330078125, 0.849609375, 1.7451171875, 0.49560546875, 1.416015625, -0.92919921875, -1.5810546875, -1.1591796875, 0.1478271484375, 0.21875, 0.54296875, 2.048828125, -0.501953125, 0.293212890625, -0.52392578125, 0.63134765625, 0.66943359375, 0.609375, -0.1702880859375, -0.2000732421875, -1.4853515625, 0.310791015625, 0.98388671875, -0.65283203125, -0.378173828125, -0.10723876953125, -1.05078125, 0.638671875, 0.429443359375, 0.88232421875, -1.08203125, 0.181396484375, 0.55615234375, 0.5703125, -1.0400390625, 1.7763671875, -0.82421875, -0.0178375244140625, -1.12109375, 1.3271484375, -0.76806640625, 1.6533203125, 0.45068359375, -0.88037109375, 0.6181640625, -0.6884765625, -0.68994140625, -0.87890625, 0.16162109375, -0.4814453125, -0.5283203125, 2.123046875, 0.52001953125, 0.62744140625, -1.5244140625, -1.0400390625, -0.2646484375, 1.4716796875, 0.697265625, 1.02734375, -0.6279296875, -0.03515625, -0.3359375, -1.1591796875, -0.195556640625, 1.31640625, 0.6689453125, 2.169921875, 0.8095703125, -1.986328125, -0.420654296875, 1.5458984375, -0.89892578125, 0.4208984375, 0.98291015625, -2.01953125, -0.1680908203125, -0.357177734375, -0.3955078125, 0.51025390625, -0.09033203125, -0.01275634765625, -0.252685546875, 0.66650390625, -0.53759765625, 0.00432586669921875, 0.71875, -0.1890869140625, 0.64794921875, -0.5712890625, 0.425048828125, -0.56298828125, -1.359375, 0.76220703125, 0.583984375, -1.1806640625, -0.634765625, -1.32421875, -1.224609375, -0.254150390625, -1.009765625, -1.1845703125, -0.7861328125, -0.39208984375, 0.194091796875, 0.221435546875, 0.5048828125, 1.7841796875, -0.560546875, 0.401123046875, -0.310791015625, 0.21044921875, -0.7470703125, -0.362548828125, -0.1253662109375, 1.3564453125, 0.1851806640625, 0.379150390625, 0.65283203125, 0.5126953125, -1.236328125, -0.267578125, -0.426513671875, -0.5498046875, 1.7529296875, -1.3525390625, -0.818359375, -0.611328125, 0.79443359375, -0.353515625, -0.98779296875, -1.5302734375, 1.15234375, 0.5263671875, -0.1396484375, -1.0517578125, -2.607421875, -0.034423828125, 0.5869140625, -0.350341796875, -1.1865234375, 0.72265625, 0.1192626953125, -0.3828125, 0.1470947265625, -0.490234375, -1.982421875, -1.6748046875, -1.1103515625, -0.82275390625, -0.607421875, -1.1611328125, 0.63623046875, -0.6015625, -0.306884765625, 0.29638671875, -0.736328125, -1.484375, -1.232421875, 1.021484375, -0.1683349609375, -0.1568603515625, -1.537109375, 0.52490234375, 0.482177734375, 0.35205078125, -0.990234375, 1.86328125, -1.0810546875, -0.85546875, -1.20703125, -0.61962890625, 0.1474609375, 0.18603515625, 1.525390625, -0.2384033203125, -0.295166015625, 0.63818359375, -0.662109375, -0.759765625, -0.76025390625, -0.6220703125, -0.9501953125, 1.228515625, -1.0927734375, -1.8037109375, -1.0087890625, 2.986328125, -0.646484375, 0.51806640625, 0.72802734375, 0.74462890625, -0.5048828125, 0.113037109375, 0.278564453125, -0.72998046875, 1.4912109375, -0.09417724609375, 0.363037109375, -0.331298828125, 0.359619140625, 0.0972900390625, 0.919921875, 0.1846923828125, -1.1279296875, 0.681640625, 0.5859375, -1.919921875, 0.259765625, -0.43359375, -1.1376953125, 1.2919921875, -0.8642578125, -1.60546875, -0.209716796875, 0.806640625, -0.166015625, 1.11328125, 0.79248046875, 0.9873046875, 0.291259765625, 0.1278076171875, -1.2744140625, -2.1171875, -0.6318359375, 0.38818359375, -0.07659912109375, -1.658203125, -0.344482421875, -0.85009765625, 0.2939453125, -6.633996963500977e-05, 1.00390625, 1.1376953125, 0.23681640625, 0.308349609375, 0.52490234375, -0.58984375, 0.904296875, -0.8359375, -0.31640625, 0.09808349609375, -1.783203125, 0.3603515625, 1.3466796875, 1.58203125, -0.2259521484375, 0.12420654296875, 0.04656982421875, -0.062347412109375, -0.0947265625, 0.2900390625, 0.1495361328125, 0.06781005859375, -0.290283203125, -0.4423828125, -1.919921875, -1.0478515625, -0.38427734375, -0.95703125, -0.81494140625, -0.822265625, -0.3251953125, 0.32958984375, 0.381103515625, -0.544921875, -0.5732421875, 0.89306640625, -0.429443359375, 0.11541748046875, 0.71337890625, -2.126953125, -0.7646484375, 2.076171875, 0.4716796875, -0.46533203125, -1.4697265625, -1.6748046875, 0.04180908203125, -1.919921875, -1.107421875, 1.890625, 0.41259765625, -0.8603515625, 1.083984375, -0.0167694091796875, 0.7490234375, 0.20849609375, 1.5927734375, -0.0701904296875, -1.44140625, -2.3828125, 0.53369140625, 1.388671875, 0.384521484375, 0.38623046875, 1.5107421875, -1.3779296875, -1.033203125, -0.8681640625, 1.06640625, 1.6748046875, 1.2001953125, -0.1063232421875, -1.669921875, -0.224853515625, -0.11407470703125, 0.6162109375, -0.166748046875, 1.43359375, 0.0109100341796875, -0.45849609375, 0.61083984375, 1.578125, -0.09619140625, -1.4443359375, -0.92724609375, 0.374267578125, 0.1607666015625, 0.763671875, -1.9033203125, 0.1553955078125, -0.64697265625, 0.05377197265625, 0.347412109375, 0.2049560546875, -2.810546875, 0.68115234375, 1.7412109375, 0.00461578369140625, -2.240234375, 0.0390625, 0.2041015625, -1.4208984375, 0.876953125, 0.43994140625, 0.28076171875, -1.166015625, -0.10089111328125, -1.2529296875, 0.734375, 0.765625, -0.09783935546875, 0.818359375, -1.9150390625, 1.095703125, 0.76513671875, 1.6015625, -0.99658203125, 1.083984375, -0.254150390625, -0.389404296875, -0.468505859375, 0.1090087890625, 0.52392578125, 0.55224609375, -0.116455078125, -0.57080078125, -0.88232421875, -0.294677734375, -0.176025390625, -0.352783203125, 0.426513671875, -0.3994140625, 0.82568359375, -1.29296875, -0.63720703125, -1.7744140625, -0.63525390625, -0.83544921875, -0.019073486328125, -0.49560546875, 0.6796875, -0.70703125, 0.92919921875, -0.53759765625, 0.3623046875, 0.4482421875, 0.1063232421875, 0.036346435546875, -0.237060546875, -0.498046875, 0.4326171875, -1.1611328125, 1.7451171875, -1.5419921875, 0.12335205078125, 1.0673828125, 1.0224609375, 0.1668701171875, 1.3955078125, 0.8310546875, -0.46484375, -0.343505859375, 0.53125, -0.24365234375, 1.2216796875, -2.345703125, 2.107421875, -0.494140625, -0.8349609375, -0.3642578125, 0.93701171875, -1.04296875, 1.5009765625, -0.480712890625, 0.91748046875, -0.304931640625, 0.281982421875, -2.712890625, -0.313720703125, -0.9931640625, -0.271728515625, 1.201171875, -0.5078125, 0.2822265625, 0.1878662109375, 0.0161590576171875, 0.6650390625, -0.260498046875, 0.697265625, -1.015625, -1.08203125, -1.0224609375, 0.57666015625, -0.41357421875, -1.4638671875, 2.537109375, -0.68115234375, -0.29296875, -0.83056640625, 0.257080078125, 1.5078125, 1.8193359375, 0.09478759765625, 0.27392578125, 1.359375, 0.5537109375, 0.25439453125, -1.7119140625, -0.18603515625, -0.305908203125, -1.4326171875, -0.0071563720703125, 1.126953125, -1.3974609375, -3.07421875, -0.71533203125, 1.19921875, -0.966796875, 0.02484130859375, -0.042694091796875, -0.5615234375, 0.1195068359375, -1.0087890625, 0.355712890625, 0.303955078125, 0.254150390625, 0.365478515625, -1.7880859375, -1.416015625, 1.693359375, -0.1148681640625, -1.052734375, 1.3515625, 0.7841796875, 0.142578125, 0.72705078125, 0.2001953125, -1.591796875, -0.4052734375, 0.2470703125, 0.5048828125, -0.27490234375, -0.64453125, -0.327880859375, -1.6533203125, 0.257080078125, -0.6162109375, -0.32666015625, -0.2269287109375, -1.255859375, 0.2080078125, -0.5146484375, -0.10894775390625, 1.169921875, 0.288330078125, 0.008209228515625, -0.09649658203125, -0.76416015625, 0.59765625, -1.2236328125, -1.189453125, 0.06854248046875, -0.9248046875, 0.0606689453125, -0.11053466796875, -2.154296875, -1.5888671875, -1.6533203125, 0.36474609375, -0.27978515625, 2.138671875, -0.6591796875, -0.72265625, -1.9443359375, -0.5703125, -1.091796875, 0.8154296875, 0.88916015625, -0.9794921875, 2.51953125, 0.0242767333984375, 1.15625, -0.2357177734375, -0.2081298828125, 0.282958984375, -0.3134765625, 0.180908203125, -0.89306640625, -2.37109375, -0.32568359375, 2.80078125, 0.45068359375, 1.3984375, -0.37744140625, -0.97314453125, 0.40771484375, -1.8017578125, 1.619140625, -0.380126953125, -0.32861328125, 0.1988525390625, 0.92138671875, -0.6357421875, 0.92724609375, 0.91455078125, -1.4560546875, 0.626953125, 1.421875, -0.2281494140625, -1.078125, 0.51025390625, 1.6240234375, -1.0732421875, 0.701171875, 1.33984375, 0.72802734375, 0.9365234375, -1.345703125, 0.6748046875, -0.2288818359375, 1.0205078125, -0.498779296875, 0.337158203125, 0.98779296875, 0.062744140625, -0.004650115966796875, -0.193603515625, 0.67138671875, 1.1875, -0.1688232421875, -0.385009765625, 0.865234375, -1.083984375, 0.05999755859375, 0.50341796875, 0.060455322265625, -0.78515625, -1.8779296875, 0.299072265625, -1.8271484375, -1.455078125, 1.3056640625, -0.47314453125, -0.379150390625, 1.373046875, -0.51708984375, -0.6728515625, 1.900390625, 0.60986328125, 0.59033203125, 0.56982421875, 0.69140625, 1.2294921875, -1.0205078125, -1.1376953125, -0.06927490234375, 0.5029296875, 0.2371826171875, -2.119140625, 0.52734375, 1.3720703125, -0.1683349609375, -0.322509765625, 0.064208984375, -0.187255859375, 0.50537109375, -0.2369384765625, -0.51171875, -0.479736328125, -0.3154296875, -0.78076171875, -0.200927734375, -0.2298583984375, -0.162109375, 0.74365234375, -0.12261962890625, -0.51806640625, -0.69482421875, 1.1474609375, -1.650390625, -0.5302734375, 1.1513671875, 0.15771484375, -0.92919921875, -0.007289886474609375, 0.1883544921875, 0.336669921875, -0.7294921875, 0.56298828125, 0.88427734375, 0.86767578125, 0.435791015625, -1.0966796875, -0.77197265625, 1.1552734375, -0.06646728515625, -0.71630859375, -0.375244140625, -0.06689453125, 2.779296875, 1.484375, -0.5810546875, -0.494873046875, -0.2012939453125, -0.11895751953125, -0.50390625, 1.0693359375, 0.82177734375, 0.2919921875, -0.548828125, -1.482421875, -0.40283203125, -0.95849609375, -0.482421875, 1.2568359375, 0.1695556640625, 0.22216796875, -0.62451171875, -1.3427734375, -1.60546875, -0.454833984375, 0.8212890625, 0.4443359375, 1.8095703125, 0.6376953125, -0.90966796875, 0.45947265625, 0.392822265625, -1.015625, 0.240478515625, -0.8515625, 1.55078125, -0.94189453125, 0.6865234375, 0.53271484375, 1.1787109375, 0.127685546875, -0.9599609375, 0.61865234375, 0.509765625, -1.12109375, -0.712890625, 1.7080078125, -0.389404296875, 1.11328125, 1.2578125, 1.7421875, -0.93701171875, 3.171875, 0.2012939453125, -0.12152099609375, -0.41845703125, 1.26953125, -0.447265625, 0.2015380859375, 0.6416015625, 1.5322265625, 1.2919921875, 2.03125, -1.21484375, -2.01953125, -0.8154296875, 0.357177734375, 0.99169921875, 0.30810546875, -0.33935546875, -0.1063232421875, -0.32421875, 0.98388671875, -2.1171875, -0.216552734375, 2.0390625, -0.0002532005310058594, 0.8740234375, 0.703125, -0.27587890625, -0.47802734375, 0.01515960693359375, -0.4150390625, 1.3427734375, -0.75537109375, -1.8759765625, -0.055999755859375, 0.52685546875, 1.0693359375, 0.142822265625, 2.8046875, 0.2437744140625, -0.689453125, 1.11328125, 1.0771484375, -0.76025390625, -2.052734375, 0.6328125, 0.66015625, 0.5185546875, -0.92578125, 0.52783203125, 1.162109375, -1.62890625, 1.67578125, 0.366943359375, 0.3125, -1.767578125, -0.4091796875, -0.69384765625, 0.408447265625, 0.8359375, -1.4775390625, -0.08453369140625, 0.207275390625, 0.4853515625, -0.65673828125, -0.20654296875, 2.3515625, -0.88232421875, 0.293212890625, -1.0400390625, -0.3720703125, 0.6474609375, 0.375, 0.71533203125, 0.44482421875, -1.4326171875, -0.6669921875, -0.9072265625, -0.6884765625, -0.18798828125, -0.7197265625, 0.86279296875, -0.354736328125, -0.420166015625, -0.23095703125, -0.5615234375, 0.5927734375, -0.798828125, 1.3271484375, -0.93505859375, 1.423828125, -0.96435546875, -1.4453125, -0.5244140625, -0.29443359375, 1.0087890625, 1.0693359375, 1.810546875, -0.61279296875, -0.59326171875, 1.3369140625, 0.51220703125, -0.400146484375, -0.0202789306640625, -0.06365966796875, -1.8251953125, -0.12451171875, -1.4287109375, -1.126953125, -0.7314453125, -0.9638671875, -0.76025390625, 0.982421875, -0.1435546875, 0.57373046875, -0.23095703125, 1.0556640625, -0.30615234375, -1.533203125, -0.053863525390625, 0.0751953125, -0.3720703125, 0.19873046875, -0.2147216796875, 1.0224609375, -0.611328125, -0.7978515625, 1.9833984375, 0.3076171875, -0.5380859375, -0.224853515625, 0.55615234375, 1.65234375, 0.031890869140625, -0.35302734375, -1.365234375, -0.77880859375, -0.0728759765625, -0.67578125, -0.08929443359375, 0.2255859375, 0.322265625, -0.896484375, 1.603515625, -0.64453125, -1.41796875, 0.039154052734375, 0.5693359375, -1.0380859375, -1.1474609375, 0.826171875, 0.57080078125, 1.7490234375, -2.47265625, 0.453857421875, 1.8173828125, 0.6845703125, 0.2392578125, 0.170166015625, 0.26708984375, 1.1025390625, 0.55810546875, -0.763671875, 0.201904296875, 0.1090087890625, -0.048065185546875, 0.302978515625, -1.1748046875, -1.2431640625, 0.69921875, -0.2376708984375, 0.77294921875, -0.90087890625, -0.689453125, -2.46875, 0.77490234375, -1.3154296875, 0.385986328125, -0.376708984375, -0.64453125, 0.63330078125, -0.60888671875, 0.2379150390625, 1.5693359375, 0.19970703125, 0.2646484375, 0.387939453125, -1.185546875, 0.127685546875, 0.86083984375, -0.260986328125, 1.1845703125, -0.89404296875, -1.1044921875, -2.642578125, 0.6357421875, -0.529296875, -0.88232421875, -1.126953125, 0.471435546875, 0.230712890625, -0.25537109375, -0.339599609375, -0.395751953125, -1.435546875, -1.9326171875, 1.2822265625, -1.087890625, -0.6591796875, 0.447021484375, -0.11773681640625, -0.9765625, 0.71435546875, -0.4404296875, -0.50439453125, 0.5595703125, 1.0390625, 0.86962890625, -0.84912109375, -0.60888671875, 0.82763671875, -0.446044921875, 0.362060546875, -0.708984375, 1.0458984375, 0.1396484375, -0.417724609375, 0.12249755859375, 0.77294921875, 0.328125, -1.2509765625, 0.9462890625, -1.962890625, 1.58984375, -0.66162109375, 1.630859375, -0.9677734375, 1.521484375, -1.2578125, 1.869140625, -1.4951171875, -1.1494140625, 0.767578125, 1.255859375, -0.6142578125, 0.25244140625, 0.96484375, -0.06549072265625, -0.59716796875, 0.06646728515625, -0.6884765625, -0.1954345703125, 0.0804443359375, 0.444091796875, 0.69677734375, 0.02166748046875, -0.380859375, -0.36181640625, -0.77685546875, -0.484130859375, -1.5283203125, -2.07421875, -0.1409912109375, -0.9833984375, -0.99609375, 1.123046875, -0.30517578125, 0.5078125, 1.0068359375, 1.2646484375, 0.578125, -0.3154296875, 3.3203125, 1.1494140625, 1.884765625, 0.943359375, -0.31982421875, 0.2391357421875, 1.1943359375, -0.45947265625, -0.2489013671875, -1.26171875, -0.30908203125, 0.259765625, 0.08123779296875, 0.41845703125, -0.496826171875, -0.5703125, 0.0447998046875, -0.5068359375, 0.371337890625, 1.21484375, 2.01953125, 0.35791015625, 0.487548828125, -0.12396240234375, -0.016326904296875, 1.4560546875, 0.83251953125, -1.7431640625, 0.79736328125, -1.396484375, -0.7236328125, 0.1529541015625, -1.431640625, -0.36083984375, -0.56396484375, -1.259765625, -0.35498046875, -1.154296875, 0.5087890625, 2.51171875, -0.1871337890625, -1.87109375, -0.69287109375, -1.287109375, 0.052490234375, 1.0830078125, 0.499755859375, 0.0859375, 2.451171875, -0.315673828125, -0.8671875, 1.953125, -2.55859375, 0.285888671875, -0.76220703125, 0.375732421875, -0.72265625, -1.1494140625, -1.931640625, -1.4248046875, -0.65576171875, -0.98876953125, -2.123046875, 0.0019989013671875, -2.0625, -1.3388671875, 0.342529296875, -1.8095703125, 0.430908203125, -1.9072265625, -1.6103515625, 1.64453125, -1.1064453125, -0.51025390625, 0.07684326171875, -1.1474609375, -0.97216796875, -0.0292510986328125, -0.6455078125, 0.482421875, 1.2412109375, 1.2099609375, 2.232421875, -1.7099609375, 0.462890625, -0.7041015625, -2.37890625, 1.3359375, -1.04296875, -0.391845703125, 1.1064453125, -1.40234375, -0.250244140625, 0.5068359375, 1.158203125, 0.343017578125, -1.134765625, -2.0546875, -0.62890625, -0.34130859375, 0.319580078125, 0.428466796875, 0.91796875, 0.03887939453125, 1.4638671875, -0.7763671875, -0.59912109375, 0.6611328125, 1.0361328125, -0.60595703125, 1.0361328125, -0.252197265625, -1.091796875, -0.2919921875, -0.130126953125, -1.3193359375, 0.307373046875, -0.951171875, -0.448486328125, 1.03125, -1.7587890625, 0.451904296875, -0.67333984375, -0.6943359375, 1.0703125, 0.9296875, -0.12384033203125, -0.2000732421875, 1.333984375, -1.046875, -1.6025390625, 1.150390625, -0.75830078125, -0.79638671875, -1.318359375, -1.1728515625, 2.041015625, -0.2271728515625, 0.515625, 0.3603515625, 1.9853515625, 0.58935546875, 1.650390625, 0.264404296875, -1.0341796875, 0.51611328125, 0.203857421875, 0.0255584716796875, -0.55322265625, 1.1474609375, 0.8798828125, -0.1673583984375, 0.4404296875, 1.6201171875, -0.78369140625, 0.320068359375, 0.29541015625, -0.2393798828125, 0.34716796875, -1.0986328125, -0.1820068359375, 0.428955078125, -1.251953125, 0.4765625, 1.2099609375, 0.489990234375, -0.405517578125, -0.2288818359375, 0.79296875, -0.28564453125, -0.1982421875, -0.70361328125, -0.55517578125, -0.5400390625, 0.195556640625, -0.75732421875, -0.47607421875, 0.2437744140625, 0.6376953125, 0.27099609375, -1.7529296875, -0.76171875, -0.84228515625, 0.45751953125, 1.37109375, -0.87646484375, 0.39990234375, -1.41015625, -0.75927734375, 0.87646484375, -0.07965087890625, -0.022125244140625, -0.332275390625, 0.61376953125, -1.486328125, -0.71240234375, -2.3828125, 0.78173828125, -0.9560546875, -1.4228515625, 0.21240234375, -1.693359375, 0.3525390625, 0.01529693603515625, 1.54296875, 0.58447265625, 0.65576171875, 1.55078125, -1.451171875, 1.2705078125, -2.5078125, 1.126953125, -0.069091796875, -0.724609375, 1.03515625, 1.328125, 0.92236328125, -0.58544921875, -0.65283203125, -1.029296875, -0.1610107421875, -0.2578125, -0.6083984375, -0.3232421875, -0.243408203125, -1.146484375, -0.112060546875, 2.052734375, -0.8427734375, -0.0262603759765625, 0.341064453125, 0.364501953125, 0.09942626953125, -0.78076171875, 0.41015625, -0.48291015625, -0.5869140625, 0.8759765625, 0.5078125, -0.54248046875, -0.06890869140625, -0.97607421875, 0.1260986328125, 1.28515625, -0.1932373046875, -0.71142578125, -0.438720703125, 1.568359375, -0.410400390625, -0.873046875, 0.2454833984375, -0.10784912109375, 1.12109375, -0.92333984375, -0.98583984375, -0.448486328125, 0.5595703125, 0.017242431640625, 0.38623046875, 0.435791015625, 1.0810546875, 0.6796875, 0.48974609375, 0.09259033203125, -0.0144195556640625, -0.77001953125, -1.283203125, 1.2529296875, 1.0166015625, -1.9736328125, -0.61474609375, 1.4912109375, -0.72802734375, 0.408203125, 0.9287109375, 0.1494140625, -0.47119140625, -1.3203125, 0.658203125, 0.662109375, 0.98486328125, -0.1337890625, -1.6650390625, 0.177734375, -0.88427734375, -1.654296875, -0.2486572265625, 0.1534423828125, -1.1396484375, 0.8388671875, -0.1669921875, -0.0511474609375, 0.26416015625, -0.028961181640625, -1.228515625, 0.6044921875, 0.619140625, -0.09075927734375, -0.6181640625, -1.234375, -2.47265625, 0.755859375, -0.73046875, 0.83837890625, 0.55224609375, -0.7421875, 0.91943359375, -0.74560546875, 1.033203125, -0.451171875, 2.201171875, 0.9375, -0.76904296875, 1.484375, -1.9462890625, 1.81640625, 0.283935546875, 0.2283935546875, 0.427001953125, 1.3798828125, -0.33251953125, 0.46826171875, 0.416259765625, 1.0966796875, -0.11981201171875, 1.830078125, -0.599609375, -0.359619140625, -2.0546875, -1.3232421875, -0.426513671875, 1.181640625, -1.0029296875, 0.10662841796875, -0.17578125, -0.345703125, 0.46533203125, -0.4638671875, -2.509765625, -0.1746826171875, 0.146240234375, -0.57666015625, -0.373046875, -0.83544921875, -0.291015625, -0.8232421875, 0.494384765625, 0.43994140625, 1.3076171875, 0.446044921875, 0.214599609375, 1.166015625, -1.0546875, -1.046875, -0.666015625, -2.16796875, -0.33203125, 0.11639404296875, 0.3212890625, -0.9833984375, -0.66357421875, 0.765625, -0.791015625, -0.29248046875, 1.814453125, -0.9755859375, -0.67431640625, 0.7421875, -0.11981201171875, -2.703125, -1.697265625, 0.73046875, 0.6083984375, -2.203125, 1.6796875, 0.77783203125, 0.072265625, 0.392333984375, 1.2109375, 2.73828125, 1.212890625, -0.38134765625, -0.91162109375, -0.55712890625, -0.98046875, -0.71435546875, -2.857421875, 0.467041015625, 0.94677734375, -0.58154296875, -0.0670166015625, -1.0400390625, 0.50927734375, 0.2288818359375, 0.1339111328125, 0.80615234375, -1.5556640625, 0.212890625, -0.405517578125, -1.8134765625, -0.9560546875, 0.6103515625, 0.62939453125, -1.4287109375, 0.31689453125, 0.677734375, -0.09033203125, -0.66845703125, 1.17578125, -0.1922607421875, 1.919921875, -1.220703125, -0.07586669921875, 0.405029296875, -2.248046875, -1.388671875, -1.69921875, -0.1416015625, 1.2216796875, -1.349609375, 0.1494140625, 0.65234375, 0.25439453125, -1.19921875, -0.281494140625, 0.70654296875, 2.67578125, 1.83203125, 0.98583984375, -2.41796875, 0.5517578125, 0.7529296875, -0.96826171875, 0.135498046875, 0.2044677734375, -0.425048828125, -0.2403564453125, -0.935546875, -0.258056640625, -0.274169921875, -0.54736328125, -0.853515625, -2.2890625, 2.810546875, 1.046875, 0.08966064453125, -0.5947265625, -0.29248046875, 1.87890625, 0.5390625, -0.54638671875, -1.552734375, -2.5390625, -1.439453125, -0.9609375, -0.264892578125, -0.54150390625, -0.277099609375, 0.50732421875, -1.0771484375, 1.2783203125, 1.4501953125, 0.64990234375, -0.6708984375, -3.408203125, 0.56591796875, -0.81494140625, 1.0634765625, 1.33984375, 1.7666015625, -0.9248046875, -0.149658203125, 0.283935546875, 0.29736328125, -0.2396240234375, 1.9189453125, -1.1689453125, -0.027313232421875, 0.1865234375, -1.58203125, 1.1787109375, -0.54638671875, 0.20703125, -0.283203125, 0.499755859375, 1.0830078125, 0.3994140625, 0.5810546875, -0.77197265625, -0.62939453125, -0.794921875, -0.317626953125, -0.68798828125, -0.1927490234375, 0.391357421875, -1.611328125, -0.259521484375, 0.806640625, -1.3369140625, 0.0726318359375, 0.316650390625, 0.6298828125, -3.177734375, -0.239013671875, 1.962890625, 1.2177734375, 0.050323486328125, -1.146484375, -0.3349609375, -0.2822265625, -1.0849609375, -1.3427734375, -1.875, 0.60009765625, -0.591796875, 0.80859375, -0.31591796875, 1.03515625, 0.287353515625, 0.654296875, 0.5234375, 0.67431640625, -0.82080078125, -1.4931640625, 0.1414794921875, 1.0146484375, 0.71044921875, 1.1796875, -0.3193359375, -1.1943359375, -0.243896484375, 1.236328125, -2.447265625, -0.262939453125, 1.87890625, 1.5849609375, 0.1094970703125, 1.7421875, -0.78564453125, -0.78125, -0.74365234375, 0.23291015625, 0.52587890625, 1.4248046875, -1.3193359375, 0.065673828125, 1.6044921875, -0.333251953125, 1.4345703125, 0.237060546875, 0.97412109375, -0.342529296875, 0.814453125, -1.322265625, 1.4619140625, -0.2177734375, -0.662109375, 0.58203125, 1.6123046875, 2.087890625, 0.2344970703125, -0.5048828125, 0.11822509765625, 0.9501953125, 0.6083984375, 1.345703125, 0.96044921875, -0.189453125, 0.46240234375, -1.505859375, -1.06640625, 0.50634765625, -0.164794921875, -0.221923828125, 1.5732421875, -0.56396484375, 0.4091796875, 1.0703125, 1.2392578125, -0.07379150390625, 0.45361328125, -0.68017578125, -0.69140625, 0.62744140625, 0.9033203125, 1.7021484375, 1.20703125, -0.6171875, 1.0966796875, 0.5166015625, 0.048309326171875, 0.5791015625, 1.0205078125, 1.2041015625, -0.28369140625, -1.54296875, 0.05987548828125, 1.552734375, 1.416015625, 0.28271484375, 1.515625, -1.5380859375, 0.357421875, -1.599609375, -0.053314208984375, -0.61328125, -0.4931640625, 0.38232421875, 0.435791015625, 2.115234375, 0.17626953125, -0.10736083984375, 1.0615234375, 0.478759765625, 0.734375, -0.73486328125, -0.81787109375, 0.17919921875, -0.3193359375, -1.263671875, -0.8662109375, 0.2763671875, -0.10577392578125, 0.92138671875, 1.0029296875, -0.6640625, 0.5478515625, 1.947265625, -0.7666015625, 0.344970703125, 0.39404296875, -1.267578125, -0.90380859375, -0.262451171875, 0.54248046875, 0.7119140625, 0.5703125, 0.359619140625, -0.2493896484375, -0.30517578125, 1.7646484375, 0.2276611328125, 0.72509765625, 1.1435546875, -0.208251953125, 0.060150146484375, 1.3583984375, -0.79052734375, 0.39990234375, -1.513671875, 0.66796875, -0.179443359375, -0.351806640625, 0.8935546875, -0.30517578125, 0.052978515625, 0.046112060546875, -0.39794921875, 1.3486328125, -0.55908203125, 0.208740234375, 1.0341796875, -1.6943359375, -0.348876953125, -1.3466796875, 1.017578125, -0.047271728515625, -0.92919921875, -0.49365234375, 1.103515625, 0.99267578125, -0.142822265625, -1.078125, -0.487060546875, -1.05078125, -0.0889892578125, -1.1015625, -1.5634765625, 1.1357421875, -0.97216796875, 0.310791015625, -0.68896484375, -0.0132904052734375, 1.1064453125, -1.1416015625, 0.307861328125, -1.994140625, -1.041015625, -1.5322265625, 0.1666259765625, -0.0740966796875, 1.5830078125, 0.62060546875, 1.2841796875, 0.70361328125, 0.0860595703125, 1.013671875, 0.1768798828125, -0.031341552734375, -0.5751953125, 0.72705078125, 0.50341796875, 1.1171875, 0.1561279296875, 0.141845703125, -0.63623046875, 0.884765625, 0.5625, 0.48583984375, 0.446533203125, 0.054229736328125, -1.173828125, 0.58740234375, 0.1693115234375, 0.70166015625, -0.96337890625, 0.73779296875, 1.490234375, 0.196044921875, 0.444091796875, -0.10986328125, 0.285400390625, 1.27734375, 1.1650390625, -0.64697265625, 0.9306640625, -0.751953125, -0.5810546875, -0.03326416015625, -1.4873046875, 0.6591796875, -1.6025390625, 0.00531768798828125, 0.74462890625, 0.01413726806640625, 1.2939453125, 1.037109375, 1.2060546875, 0.426025390625, 1.3134765625, 0.465576171875, 0.42578125, 2.443359375, -0.64306640625, -0.271728515625, -0.90185546875, 0.740234375, 0.7080078125, 0.7314453125, -0.78173828125, 1.65234375, 0.268310546875, 0.66943359375, 0.0364990234375, 0.82958984375, 1.1357421875, -0.34423828125, 0.213134765625, 0.2119140625, -2.068359375, -0.1922607421875, 0.7919921875, -0.56201171875, 0.2900390625, -0.07366943359375, -0.91064453125, 0.88720703125, 0.1279296875, -1.66015625, -0.29248046875, -0.1226806640625, -0.70068359375, 0.8779296875, -0.67822265625, -0.43115234375, 0.421875, 1.03125, 0.218994140625, -0.7880859375, 0.61767578125, 1.208984375, 0.00441741943359375, 1.310546875, 1.7890625, 1.375, 0.93310546875, -1.119140625, 0.57275390625, 1.4345703125, 0.32763671875, -0.358642578125, 0.2841796875, 0.97021484375, -1.2529296875, -0.28466796875, 0.1741943359375, -0.275390625, -0.223876953125, -0.943359375, -0.8564453125, -1.2001953125, -0.406982421875, 1.4443359375, 2.095703125, -0.15283203125, 1.7685546875, -0.467529296875, -1.337890625, 2.794921875, 0.04486083984375, -0.45654296875, -1.0380859375, -0.65576171875, -0.5771484375, 0.68408203125, -0.01053619384765625, 1.5302734375, -0.32275390625, 0.57080078125, 0.4833984375, 0.95654296875, -0.68017578125, -0.56005859375, -0.333740234375, 1.046875, 0.80810546875, -0.2471923828125, 0.034912109375, -2.15625, -0.7802734375, -0.03399658203125, -0.76318359375, -0.258544921875, 0.37158203125, -0.79541015625, -0.40869140625, 1.02734375, -1.248046875, 0.8212890625, -1.0576171875, 0.2493896484375, -1.2568359375, -0.380615234375, -1.3427734375, 0.96484375, 0.0841064453125, 0.3818359375, -0.493896484375, -1.171875, 1.041015625, 0.47314453125, 1.6064453125, 0.1859130859375, 0.197998046875, -0.578125, 0.548828125, 1.2080078125, -0.45556640625, -0.6572265625, 0.50341796875, -0.47607421875, 0.75439453125, 0.93896484375, -0.461181640625, 0.90576171875, 0.796875, -0.1771240234375, -1.3447265625, 1.908203125, -0.8583984375, -0.057373046875, 0.43701171875, 0.81982421875, 0.07745361328125, 1.3037109375, -1.6474609375, -0.494384765625, 0.2138671875, -0.05914306640625, -1.2763671875, 0.408203125, 0.1829833984375, 0.91748046875, 0.202880859375, 0.8857421875, 0.72265625, 0.128173828125, -0.1031494140625, 0.8427734375, 1.056640625, -0.27783203125, -0.81689453125, 2.25390625, -1.2685546875, 0.228515625, -0.1495361328125, -0.86328125, -1.119140625, -0.039581298828125, 0.669921875, 1.109375, -0.3583984375, 0.93505859375, 1.96875, -0.8017578125, -0.447998046875, -0.08782958984375, -0.939453125, -1.302734375, 0.7412109375, -1.345703125, -0.037933349609375, -0.57958984375, -0.833984375, -1.23828125, 0.87255859375, -0.33544921875, 0.90283203125, 0.1800537109375, -1.01171875, 0.421875, 0.7177734375, 1.0908203125, 2.125, 0.172119140625, 0.5712890625, 0.77783203125, 0.505859375, 1.7724609375, -0.58203125, -0.7109375, 0.69873046875, -1.0361328125, 1.5078125, -0.59619140625, 0.87109375, -1.0107421875, 1.5888671875, 0.1729736328125, 0.037933349609375, -0.994140625, -1.1005859375, -0.390625, -0.68359375, -1.0302734375, -0.4384765625, -1.5732421875, 0.36376953125, 0.39306640625, -1.3818359375, 0.76123046875, 0.79541015625, 0.03277587890625, -0.56298828125, -0.07965087890625, -1.4150390625, -1.6142578125, -0.5498046875, -0.16015625, -1.2197265625, 0.38916015625, -0.485595703125, 0.0662841796875, 0.64697265625, 0.1578369140625, 0.52978515625, 0.5498046875, -1.2109375, 0.73779296875, 0.71875, -0.82958984375, -1.5703125, 1.01953125, -0.67626953125, -1.0107421875, 1.0107421875, 0.89208984375, -0.9208984375, -0.159912109375, -0.58349609375, -1.17578125, 0.2254638671875, -0.36669921875, 1.482421875, -2.13671875, -0.56640625, -1.328125, -0.07000732421875, -0.393798828125, 0.50634765625, 0.849609375, -0.5087890625, 0.169189453125, 1.125, -2.94140625, 0.294189453125, 0.74755859375, 0.6962890625, -0.96875, -1.408203125, 0.180419921875, -0.44677734375, 0.02484130859375, -0.7197265625, -1.1923828125, 1.630859375, -2.71484375, -0.33203125, 1.837890625, -0.1202392578125, 0.09747314453125, 0.56982421875, -0.6611328125, 0.80615234375, 1.2275390625, -0.51171875, 0.0419921875, -0.17578125, -0.04962158203125, 0.451171875, -0.5859375, -1.044921875, -0.441162109375, 2.26953125, -0.07861328125, 0.7958984375, 1.1474609375, -0.135009765625, 0.39892578125, 1.32421875, 1.0791015625, -1.935546875, 0.833984375, 0.52734375, 1.115234375, -0.505859375, -0.2666015625, -1.0625, 1.8994140625, -1.2900390625, 0.311279296875, -1.1591796875, -0.69970703125, 1.310546875, 0.254150390625, -0.240966796875, 2.998046875, -0.07525634765625, 0.79638671875, 0.065185546875, 0.95556640625, -1.0146484375, -1.8017578125, 1.58203125, 0.99951171875, -1.376953125, 0.1749267578125, -1.9248046875, -0.79833984375, -0.962890625, -0.4736328125, -0.213134765625, -0.60888671875, 1.3232421875, 0.2578125, 0.2176513671875, 2.142578125, -0.079833984375, -3.4296875, 0.36962890625, -0.332763671875, 0.01251983642578125, 0.5068359375, 0.275390625, 0.0933837890625, -0.455322265625, -0.2744140625, -0.47607421875, 0.74560546875, -0.4833984375, -0.97802734375, 0.73583984375, 2.1328125, -1.1748046875, 0.1441650390625, -0.2313232421875, 1.0986328125, -1.8798828125, 0.9091796875, -1.611328125, -0.41748046875, -1.4619140625, 0.87744140625, 1.287109375, 0.66552734375, -1.0732421875, -0.36376953125, 1.900390625, 0.822265625, -0.86376953125, 1.1435546875, -0.9853515625, -1.5576171875, -0.888671875, -0.441650390625, 0.471435546875, -2.359375, 0.019805908203125, -0.1915283203125, -0.369384765625, -1.0517578125, -0.8740234375, 1.3896484375, 0.265380859375, 0.5703125, 0.2822265625, -0.424560546875, 1.6259765625, 0.357421875, 0.01110076904296875, 1.7783203125, -0.052947998046875, 0.80615234375, -0.53662109375, -0.237060546875, 1.7587890625, 0.394287109375, 0.246337890625, 1.15625, -1.283203125, 2.41015625, 0.68310546875, -0.6123046875, -0.70361328125, 0.030731201171875, -0.480224609375, -0.319580078125, -0.02606201171875, 0.118896484375, -1.0615234375, 0.035308837890625, 1.3681640625, 0.72314453125, -0.51611328125, 0.74072265625, 0.81982421875, -1.0634765625, -0.337646484375, 0.051055908203125, 1.890625, 0.27880859375, -0.40673828125, 0.734375, 0.93359375, -1.7685546875, -1.880859375, 1.1884765625, 0.282470703125, -0.6025390625, 0.0736083984375, 0.04632568359375, -0.67626953125, 1.5859375, 0.515625, -1.193359375, 0.381103515625, -1.556640625, -1.0263671875, -0.172607421875, -0.72900390625, 0.73974609375, 1.689453125, 1.2041015625, -0.8984375, 1.1552734375, 1.017578125, -0.1396484375, 0.0243682861328125, -1.390625, 0.403564453125, 3.1328125, 0.01361083984375, 0.2841796875, 0.9443359375, -1.4423828125, 1.6416015625, 0.350341796875, -0.57470703125, 0.354248046875, -0.71435546875, -0.294189453125, -0.2193603515625, -1.1875, -0.345703125, -2.21875, 0.8818359375, 0.4013671875, 0.388916015625, -1.7666015625, -0.10650634765625, -1.654296875, 0.284912109375, 0.404296875, 0.7119140625, 0.21484375, 0.07476806640625, 2.087890625, -0.80908203125, -0.513671875, -1.4775390625, 1.248046875, 0.0293426513671875, -0.5615234375, -2.0703125, 1.67578125, 1.76171875, 1.0205078125, -0.169677734375, -1.82421875, -1.6396484375, 2.732421875, 0.19287109375, -0.458251953125, 1.4853515625, 0.52734375, -0.97998046875, -0.1151123046875, -0.33349609375, 0.64013671875, 0.0283203125, 2.212890625, -0.486328125, 0.93212890625, 0.252685546875, -2.3828125, -0.90234375, -0.9794921875, -0.297119140625, 0.5126953125, -0.51513671875, 0.2080078125, -0.038360595703125, 2.0078125, 0.6123046875, 0.64404296875, -0.52001953125, -1.39453125, 0.267822265625, 1.2412109375, -0.327392578125, 1.3232421875, -0.08056640625, 0.0159149169921875, -0.68310546875, 0.10101318359375, 0.8173828125, 0.62353515625, -0.5068359375, 1.3984375, -2.42578125, 0.6708984375, 1.3076171875, 0.8623046875, -1.017578125, 0.09552001953125, 1.689453125, -0.56689453125, 0.52099609375, 1.1767578125, -1.3994140625, -1.322265625, 1.0, -0.113037109375, -0.3095703125, 0.52880859375, -0.07464599609375, -0.1162109375, -2.49609375, -0.38720703125, 0.3349609375, 0.38232421875, -0.401123046875, -1.322265625, -1.060546875, 0.4619140625, 1.935546875, -0.99951171875, 0.59423828125, 0.35107421875, -1.78515625, 0.5498046875, -1.4033203125, 0.3818359375, 1.2392578125, -1.0576171875, -0.4375, -1.41015625, -0.927734375, 0.88427734375, -0.82275390625, 1.3291015625, 0.46923828125, 0.986328125, -1.0078125, -0.10052490234375, 0.56591796875, -1.283203125, 0.64990234375, 0.755859375, 0.60888671875, -0.465087890625, 1.216796875, -0.73779296875, -0.80615234375, 1.052734375, -0.08544921875, -0.7392578125, 0.08917236328125, -0.3203125, -1.46484375, 0.00286102294921875, 0.346923828125, -2.00390625, 0.8955078125, 0.05987548828125, -1.115234375, 0.96240234375, 1.361328125, -0.483154296875, 0.171875, -0.1326904296875, -1.6201171875, -1.2919921875, -0.384765625, -1.10546875, 0.098388671875, 0.130615234375, -0.451416015625, 0.476318359375, -1.390625, -1.18359375, -1.833984375, -1.3447265625, 1.09765625, 0.642578125, 0.312744140625, -0.01522064208984375, 0.208740234375, 2.083984375, -0.046112060546875, 0.040069580078125, 0.51416015625, -1.6455078125, 0.15771484375, -0.34423828125, 0.00817108154296875, 0.1875, -0.290771484375, -1.125, -0.46484375, 1.8330078125, -0.67138671875, 0.014068603515625, -1.2001953125, 1.912109375, -0.5166015625, 0.560546875, -1.3173828125, 0.50341796875, 1.74609375, -0.039031982421875, 0.436767578125, 1.5068359375, 1.5390625, -1.0078125, 0.2060546875, -0.77880859375, -0.72021484375, 0.278076171875, -0.171875, -0.71484375, -2.771484375, -0.24365234375, 0.16796875, 0.17578125, 0.75732421875, 1.974609375, 0.12200927734375, -0.447998046875, 1.0693359375, 0.2452392578125, -1.0771484375, -0.0928955078125, 1.73046875, 1.0615234375, 0.154052734375, 0.4658203125, -0.642578125, 0.1654052734375, -1.623046875, 0.345703125, -0.34033203125, 0.310546875, 0.051361083984375, -0.8955078125, 0.156982421875, -0.438720703125, 0.591796875, -3.4296875, -1.59765625, 1.2470703125, 1.27734375, 1.9267578125, 0.2880859375, -0.93115234375, -2.1953125, -0.00322723388671875, 0.654296875, -1.228515625, 0.962890625, -1.642578125, -1.11328125, 1.2314453125, 0.53271484375, 0.328125, 0.1649169921875, -1.10546875, 1.490234375, 0.01398468017578125, -0.74853515625, 0.6591796875, -0.01036834716796875, -0.48779296875, -0.60205078125, 0.292236328125, 0.6513671875, -1.0615234375, 0.93115234375, -1.294921875, 0.39501953125, -2.88671875, 0.304443359375, -0.2099609375, 0.39794921875, 0.83349609375, -0.80859375, -0.60791015625, 0.85009765625, 0.442626953125, -0.76416015625, -0.384765625, 0.1563720703125, -0.42822265625, 0.00568389892578125, 1.6630859375, 0.1787109375, -1.1650390625, 0.56787109375, -1.3134765625, -0.10089111328125, 0.9580078125, 0.359619140625, 0.81689453125, 1.2900390625, 0.414794921875, 1.0703125, -0.09600830078125, 0.51806640625, -0.642578125, 0.438720703125, -0.486328125, 0.5068359375, -0.326904296875, 0.313720703125, 1.2080078125, 0.5869140625, 0.5556640625, -0.10870361328125, -1.1474609375, -2.41796875, -0.51171875, 1.244140625, 1.0283203125, 0.73095703125, -1.4384765625, 0.7783203125, 0.281005859375, -1.6845703125, -1.974609375, -1.4501953125, 0.036590576171875, -0.422119140625, -0.2164306640625, -1.7099609375, 0.022613525390625, 0.0501708984375, 0.6328125, -1.818359375, -0.55322265625, 0.1884765625, 1.2490234375, 0.23388671875, -0.06756591796875, 1.173828125, -0.40283203125, 0.5244140625, -1.451171875, -0.76513671875, -1.6357421875, 0.023193359375, 0.215087890625, 0.403076171875, 1.244140625, -0.7373046875, 0.2230224609375, -0.022552490234375, 0.98095703125, -0.833984375, -1.255859375, -0.5673828125, 0.71875, -0.09918212890625, 1.796875, 1.072265625, 0.103515625, 1.138671875, -0.5068359375, -0.4443359375, 1.015625, 1.3583984375, 0.6259765625, 0.08697509765625, -0.66015625, -0.86572265625, 0.1605224609375, 0.66064453125, 1.2392578125, 0.24169921875, -0.56103515625, 1.8876953125, -1.5546875, -0.73583984375, 0.70166015625, 0.2303466796875, -0.061492919921875, -1.76953125, 0.83740234375, 1.92578125, -1.8173828125, -1.53515625, -1.587890625, 0.69091796875, 1.296875, -1.18359375, -0.65283203125, 0.167236328125, 0.2100830078125, 0.44482421875, 0.9736328125, -0.67626953125, -1.1201171875, -0.98388671875, -0.087158203125, -0.27978515625, 0.8916015625, 1.0078125, 0.199951171875, 0.089111328125, 0.5078125, 0.05517578125, 1.6650390625, -0.658203125, 0.5048828125, 1.3017578125, -0.16845703125, -0.60498046875, 0.460693359375, 1.2685546875, 1.63671875, 0.339599609375, -0.2381591796875, 1.384765625, -0.036529541015625, -0.1263427734375, 0.0706787109375, -0.87353515625, 1.23046875, 0.9697265625, 0.25390625, -1.1826171875, 0.7685546875, -0.86572265625, 2.029296875, -0.87158203125, 0.05975341796875, -0.313720703125, 0.8662109375, 0.27490234375, 0.591796875, 0.2293701171875, 1.2333984375, -1.46875, 0.07550048828125, 0.391845703125, -1.6103515625, -0.69287109375, 2.41796875, -0.291748046875, 1.6328125, 1.9560546875, 1.154296875, 1.48046875, 0.80029296875, -0.1170654296875, 1.2841796875, 0.69140625, 0.060211181640625, -0.1180419921875, 0.68505859375, -0.52392578125, -0.447998046875, 0.19140625, -0.053253173828125, -0.78076171875, -0.2337646484375, 0.059051513671875, 1.40625, 2.02734375, -0.9384765625, 0.045989990234375, -0.109130859375, -0.5078125, 0.247314453125, 0.92333984375, -0.476806640625, 2.455078125, 1.87109375, -1.505859375, 0.412109375, -0.69873046875, -2.029296875, 0.6455078125, -1.466796875, 0.6259765625, -1.6181640625, -0.93896484375, 1.8212890625, -0.11834716796875, 0.43701171875, -0.446533203125, -0.99169921875, 1.6513671875, -1.3564453125, 0.7734375, -0.036407470703125, 2.298828125, -1.068359375, -1.4169921875, 0.42041015625, -0.83154296875, 0.724609375, 0.11199951171875, -0.1605224609375, -0.97705078125, -0.428466796875, -0.06842041015625, -0.845703125, 1.892578125, -0.242919921875, -1.2373046875, -0.10137939453125, 0.58984375, 0.16650390625, 1.0654296875, 1.1015625, 0.2132568359375, 0.1605224609375, -0.78076171875, -1.9990234375, 0.5234375, 0.302978515625, -1.0556640625, -0.0977783203125, 2.884765625, -0.71044921875, 0.84228515625, 0.76708984375, 0.0107574462890625, 0.87255859375, -0.35498046875, 1.4423828125, 0.237060546875, 0.10394287109375, 0.619140625, 0.204345703125, 1.30859375, 1.2666015625, -0.59375, 0.435546875, 0.93994140625, -0.0129547119140625, 0.94677734375, -0.513671875, -0.71923828125, 0.580078125, 0.841796875, -0.64111328125, -0.2076416015625, 0.10943603515625, -0.5751953125, 0.148193359375, 1.1708984375, -0.410400390625, 0.95751953125, -0.37060546875, 1.0869140625, -0.814453125, 2.2265625, 0.517578125, 0.7314453125, -0.2255859375, -0.4404296875, 0.25439453125, -0.419921875, -0.058135986328125, 1.1474609375, 0.5927734375, -2.37890625, -0.12396240234375, -0.33447265625, -1.8515625, 1.16796875, 2.2109375, -0.163818359375, -1.05078125, 0.369384765625, -0.74072265625, -1.654296875, -0.7138671875, -1.0537109375, 0.27392578125, 0.271728515625, -1.3193359375, 0.10089111328125, 1.275390625, 2.0078125, 0.24169921875, 1.22265625, 0.9150390625, 0.802734375, -0.496826171875, -0.1220703125, 0.44384765625, 1.22265625, -0.6259765625, -0.81396484375, -1.2548828125, -0.10009765625, 0.81640625, 0.2220458984375, -1.93359375, 0.33251953125, -0.56494140625, 0.923828125, 0.249755859375, -1.359375, 0.6455078125, -1.123046875, 1.708984375, 0.259033203125, 1.4384765625, -1.0107421875, 1.0126953125, 0.226318359375, -0.475830078125, 0.92236328125, -0.78857421875, 0.29541015625, -0.315185546875, 1.23828125, -0.67138671875, -0.85498046875, 0.47509765625, -0.1904296875, -0.294189453125, -0.703125, -2.578125, 1.38671875, 0.53369140625, 0.57421875, 1.6396484375, -1.2861328125, -0.217041015625, -1.9658203125, -1.37890625, 1.1796875, 2.419921875, -0.195068359375, -1.0888671875, -1.1083984375, -0.422119140625, -1.076171875, -0.1134033203125, -1.3466796875, 0.127197265625, -1.8994140625, -1.2119140625, 0.0009355545043945312, -0.767578125, 1.064453125, 1.818359375, 0.021942138671875, 0.2044677734375, 1.318359375, -0.356689453125, 0.038665771484375, -1.1962890625, 1.099609375, -0.081298828125, 0.56494140625, 1.240234375, -0.365966796875, 1.3974609375, 0.434814453125, 1.0283203125, -0.417724609375, 0.211181640625, -0.9658203125, -0.6845703125, -1.7998046875, -0.60498046875, 0.255126953125, 0.0872802734375, -0.434326171875, -0.033447265625, -0.66455078125, -0.85302734375, 0.32080078125, -0.70458984375, -0.24365234375, -0.9228515625, 0.04193115234375, 0.68310546875, 0.165283203125, -1.7880859375, 1.0341796875, 1.681640625, -1.755859375, 0.4150390625, 0.93017578125, -0.0276641845703125, 0.29296875, -0.77490234375, 0.0181884765625, -0.482177734375, 0.53662109375, -0.6416015625, -1.35546875, 0.4853515625, 1.3896484375, 1.8427734375, 0.382080078125, -1.1123046875, -0.047576904296875, 0.77099609375, -1.546875, 0.41259765625, -0.3984375, 0.0006322860717773438, 0.23583984375, -1.705078125, 0.6669921875, -0.284912109375, -0.66357421875, -0.17236328125, 1.2001953125, 0.4873046875, -0.7314453125, 0.5673828125, -0.046966552734375, 0.372802734375, -0.135498046875, -0.350830078125, -0.2071533203125, 0.151123046875, -1.880859375, -0.970703125, 0.685546875, -0.0196075439453125, 0.67041015625, 0.341552734375, -1.564453125, -1.6845703125, 1.333984375, -0.01515960693359375, -1.1240234375, 0.005451202392578125, -0.78759765625, 0.56884765625, 0.7783203125, 2.84375, -1.9423828125, 1.90625, -1.421875, -0.316162109375, 1.4921875, 0.410400390625, 2.021484375, -0.361083984375, 0.62158203125, 0.095947265625, -0.1331787109375, -1.1904296875, 1.3046875, -0.00978851318359375, 0.94921875, -0.3486328125, -0.94873046875, -0.88427734375, -2.720703125, -0.470458984375, -0.99169921875, 2.0703125, -0.412353515625, -0.66064453125, -2.173828125, -0.477294921875, 0.1719970703125, -1.0283203125, 0.036468505859375, 0.0806884765625, 1.1650390625, 0.97900390625, 0.1988525390625, 0.47265625, -1.576171875, -0.763671875, 0.880859375, -0.76611328125, 0.375244140625, 2.263671875, -0.023529052734375, -2.294921875, 0.54345703125, -0.6494140625, 0.2474365234375, 0.94482421875, -0.3486328125, 1.1025390625, 0.2294921875, 0.15234375, 0.328857421875, -0.95849609375, -1.6513671875, 0.08349609375, 0.58642578125, -0.88134765625, 1.1826171875, 1.224609375, 0.8466796875, -0.2132568359375, 1.341796875, -0.7529296875, -0.02679443359375, 0.0028438568115234375, -1.4521484375, -1.8291015625, -0.297119140625, -0.487548828125, 0.46630859375, -1.78515625, 0.82763671875, 0.1357421875, 0.1734619140625, -1.740234375, -0.296142578125, 0.2470703125, 0.397216796875, 0.6318359375, 0.0594482421875, -0.2064208984375, 1.16796875, -0.04608154296875, -1.1767578125, -0.5888671875, -0.9765625, 0.9453125, 1.3818359375, -0.489990234375, -0.86669921875, -0.2481689453125, -0.3173828125, -1.115234375, -0.91748046875, -0.11834716796875, -0.336181640625, -0.031097412109375, 0.0654296875, 0.0731201171875, 1.08203125, -0.399658203125, 0.83984375, -0.9638671875, 0.485107421875, -1.623046875, 1.0595703125, -0.7822265625, 2.3125, 1.05859375, -0.424560546875, 0.7158203125, 0.2066650390625, -2.52734375, -0.71923828125, 0.62646484375, 1.5439453125, 0.76904296875, 0.01348114013671875, -0.049713134765625, -0.29052734375, 1.9326171875, 2.220703125, 0.53369140625, -0.52197265625, 0.62451171875, 0.73388671875, -0.92041015625, 1.74609375, -0.57666015625, 0.25146484375, -1.005859375, -1.869140625, 0.2958984375, 2.296875, 0.546875, -1.00390625, 0.513671875, -1.24609375, -0.94384765625, -0.48681640625, 0.212158203125, -0.51123046875, 0.6435546875, 0.0135955810546875, -0.31201171875, 0.69189453125, 1.1181640625, -0.7099609375, -2.763671875, -0.87255859375, 1.171875, -0.369873046875, -0.646484375, -1.4755859375, -0.21875, 0.11444091796875, 1.3515625, 1.259765625, -1.81640625, -0.798828125, -0.51220703125, -0.6015625, 0.67919921875, -1.08984375, -0.6787109375, -0.2177734375, -0.65966796875, 0.97412109375, -1.0693359375, -1.3759765625, -1.5126953125, 2.435546875, 1.796875, 0.74951171875, 0.92626953125, 0.7001953125, 0.421630859375, 0.5078125, 1.2509765625, 0.397216796875, 0.05828857421875, 0.10150146484375, -2.78515625, -1.494140625, 0.70263671875, 0.9140625, 1.662109375, 1.419921875, 0.98681640625, 0.3037109375, -1.296875, -0.5986328125, 0.084228515625, 0.0283203125, -0.1566162109375, -1.7861328125, 0.6982421875, -0.83935546875, -0.86669921875, 0.7021484375, 1.2685546875, -1.357421875, -0.751953125, 0.0021076202392578125, -0.35400390625, 1.4189453125, 0.322509765625, 1.2236328125, -0.05804443359375, -0.022430419921875, 0.75634765625, -0.95947265625, 2.341796875, -0.033599853515625, -0.165771484375, -0.63720703125, -0.5771484375, 0.6962890625, 0.0889892578125, -0.64501953125, 1.3935546875, 0.356689453125, 0.379638671875, 0.278564453125, 1.6455078125, -0.47119140625, -0.521484375, 1.3798828125, -0.6328125, 0.025177001953125, 0.383056640625, -0.99365234375, 0.46875, -0.047393798828125, -1.12109375, -0.73193359375, 0.3720703125, 1.1103515625, -1.3505859375, -0.8974609375, 1.720703125, 0.88330078125, 0.413818359375, 1.763671875, -1.443359375, 0.03826904296875, -0.43359375, 1.8427734375, -0.39501953125, -0.12408447265625, 0.52197265625, -0.441650390625, -0.0977783203125, 0.734375, 0.8759765625, 0.470947265625, -0.0020732879638671875, 0.07659912109375, 2.1953125, -0.7744140625, 0.072509765625, -1.2470703125, -0.80517578125, -0.7646484375, 0.9697265625, -0.034027099609375, -0.0694580078125, 0.81494140625, -1.431640625, 1.9228515625, -0.243408203125, 1.0009765625, 0.427490234375, -1.9189453125, 1.6123046875, 0.86572265625, -0.1265869140625, -0.2392578125, 0.83837890625, 0.1982421875, -0.5087890625, 0.8974609375, -0.91357421875, -0.62109375, -0.89990234375, 1.0615234375, 0.2166748046875, -0.9189453125, 1.1533203125, 0.421142578125, -0.728515625, 0.5185546875, 0.39306640625, 0.39794921875, -1.607421875, 0.437744140625, 0.71533203125, -0.96240234375, -0.0487060546875, 0.2425537109375, 1.021484375, 0.464111328125, -0.04852294921875, -0.391845703125, 0.115234375, 1.787109375, -1.16015625, -0.61376953125, 0.716796875, 0.62939453125, -0.9072265625, -0.489013671875, 0.67431640625, -0.0035724639892578125, -0.468994140625, 0.5693359375, 0.78125, 0.78564453125, 0.1329345703125, -1.8173828125, 0.0262451171875, -1.2705078125, 0.1624755859375, 1.826171875, -0.78759765625, -2.25390625, -0.5537109375, -0.95947265625, 1.197265625, 0.1641845703125, 0.61865234375, 1.2373046875, -1.2666015625, -2.029296875, -0.3505859375, 0.63037109375, 1.736328125, -0.4716796875, -0.23046875, -1.0615234375, 0.143310546875, -1.7255859375, -0.335205078125, -0.623046875, 0.73828125, -0.419189453125, 0.6416015625, -1.8203125, 0.284912109375, 1.1689453125, -1.271484375, 1.2080078125, -0.7529296875, -0.888671875, 0.4560546875, -0.765625, -0.94677734375, -0.75244140625, 0.6474609375, -0.6064453125, -0.1839599609375, -0.84228515625, -0.64697265625, -0.383056640625, -0.477294921875, 1.828125, 0.2890625, 0.96923828125, 0.90380859375, -0.28125, 0.71923828125, -0.80908203125, -2.30078125, -2.16796875, 0.642578125, 0.97607421875, -1.6435546875, 0.1614990234375, -1.3291015625, -1.4736328125, -2.01171875, -0.06475830078125, -0.12939453125, 0.95654296875, 0.82080078125, 0.31494140625, 0.204833984375, -1.697265625, 1.6123046875, -0.234619140625, -0.578125, 0.505859375, 2.083984375, -0.50537109375, 1.7802734375, 0.243408203125, 0.07879638671875, 0.80322265625, 0.29541015625, -1.0947265625, -0.10479736328125, 1.7734375, -1.3359375, 0.8349609375, 0.79345703125, 0.7509765625, -0.9443359375, -0.143310546875, -0.228515625, 1.05078125, 0.91162109375, 0.66064453125, 0.15869140625, 0.39453125, -1.7841796875, 0.52294921875, -0.89501953125, -0.406982421875, 0.362060546875, -0.1549072265625, 1.3701171875, 1.2392578125, -1.767578125, -1.98046875, 0.497802734375, -2.92578125, -1.1142578125, -0.1939697265625, 2.5390625, 1.3408203125, -0.98046875, -1.53515625, -0.376220703125, -0.6728515625, -0.68505859375, -1.197265625, 0.9072265625, -0.03759765625, 1.494140625, -0.050140380859375, -1.1044921875, -0.376953125, -1.8115234375, -0.388916015625, 0.7314453125, -0.1612548828125, -0.71875, 0.137451171875, -0.174560546875, -0.0180816650390625, -0.09075927734375, -1.2255859375, 1.125, -0.2413330078125, 1.1708984375, -0.51416015625, 0.78955078125, 0.6748046875, 1.9150390625, -0.560546875, -0.1998291015625, 0.48681640625, -0.9150390625, -0.293212890625, -1.9990234375, 0.9521484375, -0.9833984375, 2.28515625, 1.388671875, 0.1385498046875, 0.115234375, 0.6279296875, 2.677734375, -1.244140625, -0.666015625, 0.759765625, -1.3837890625, -0.07806396484375, 0.117431640625, -0.703125, -0.5283203125, 0.8251953125, -0.85400390625, -1.2666015625, -0.91455078125, -0.7431640625, -0.4814453125, -0.4189453125, 0.93798828125, 2.115234375, 0.4892578125, 0.2239990234375, -0.328125, 0.56591796875, -0.1319580078125, 0.9775390625, 0.0256195068359375, -1.0625, 1.763671875, -1.0390625, -0.431396484375, -2.20703125, -0.33447265625, -0.92578125, 1.6552734375, 0.142822265625, -0.0159454345703125, -1.9931640625, -0.04376220703125, 0.779296875, -0.186767578125, -0.404296875, -0.54150390625, -0.146484375, 1.884765625, 0.41162109375, 1.51953125, -0.54541015625, 1.619140625, -0.2371826171875, 2.45703125, 1.83203125, 1.3701171875, -0.5, 1.6015625, 0.410400390625, 0.56640625, -1.38671875, 0.6943359375, 1.916015625, -0.533203125, -0.170654296875, 2.154296875, 1.52734375, 1.51953125, 0.0218353271484375, 1.5703125, -1.0048828125, -0.359130859375, -2.646484375, -2.8828125, -0.2919921875, -0.6181640625, 0.25830078125, -1.01171875, 0.048431396484375, 0.978515625, 3.61328125, 1.9560546875, 0.375244140625, -0.111328125, 0.51171875, -0.4453125, 0.77587890625, -1.9423828125, -0.77392578125, -1.2490234375, 0.03289794921875, 0.77197265625, 0.892578125, -1.673828125, 0.363525390625, 1.7880859375, -0.38037109375, -0.10064697265625, 1.212890625, -0.16650390625, 1.537109375, -1.1025390625, 1.66015625, 1.1826171875, -0.9189453125, -1.9853515625, -0.353271484375, 1.037109375, 0.607421875, -1.4560546875, 0.70703125, 0.120361328125, -0.63671875, 0.07257080078125, 0.305908203125, 1.69921875, 1.318359375, 0.2451171875, -0.480712890625, -1.3388671875, -1.78515625, 0.9365234375, -0.91259765625, 1.0166015625, -0.8193359375, 0.87890625, 1.0634765625, 0.0357666015625, 0.75830078125, 2.08984375, 0.489990234375, 0.31103515625, -2.861328125, 0.07147216796875, -1.3134765625, 2.162109375, -0.97216796875, -2.671875, 0.58642578125, -1.9287109375, 0.583984375, -0.93798828125, -1.1826171875, 0.25830078125, 0.740234375, -0.78857421875, 0.12030029296875, 0.6494140625, -0.7685546875, 0.8759765625, 0.90185546875, -0.7724609375, -1.4169921875, -1.59375, 0.01043701171875, -0.986328125, 0.65185546875, -3.34375, -0.1087646484375, 0.050567626953125, 0.86181640625, 1.7421875, 0.455810546875, 0.4931640625, -0.286376953125, 0.7607421875, 0.443359375, 0.90380859375, 0.9111328125, -0.59912109375, -0.33154296875, 0.55419921875, -1.333984375, 0.300537109375, 1.24609375, -0.72509765625, -0.67626953125, 0.2137451171875, -0.84814453125, -0.424560546875, -0.5126953125, 1.0732421875, 0.373046875, 2.28125, 0.64111328125, 0.1246337890625, -0.08856201171875, -0.4765625, -0.058807373046875, -0.6083984375, -0.81787109375, 0.175537109375, 0.65771484375, -0.11566162109375, -0.039276123046875, -0.92138671875, 0.82958984375, 1.6015625, 1.3095703125, -0.6982421875, 1.681640625, 0.089599609375, -0.63427734375, 0.0809326171875, 0.3779296875, 0.9814453125, -0.055450439453125, -0.17919921875, 1.0966796875, -0.1934814453125, 0.12017822265625, 1.3564453125, 0.2412109375, 0.33349609375, 0.2489013671875, 0.93798828125, -0.264404296875, 0.373779296875, -1.1279296875, -0.358154296875, -0.337646484375, -0.2548828125, -0.0313720703125, -0.1514892578125, 0.2890625, 0.7822265625, 0.90966796875, 1.197265625, -2.0859375, -0.59228515625, 1.1943359375, -1.61328125, 0.81103515625, -1.0, 0.87744140625, 0.108642578125, 2.1796875, -0.90185546875, -0.4990234375, -0.35107421875, 0.68115234375, -0.76171875, -0.313232421875, 0.0555419921875, -0.008575439453125, 1.123046875, 0.9990234375, -0.63818359375, -1.2470703125, -0.25146484375, 0.193603515625, -1.57421875, 0.222412109375, -0.2418212890625, 2.69921875, 1.869140625, 0.351318359375, -0.61865234375, 1.3095703125, 0.521484375, -1.5068359375, 0.199951171875, 0.248779296875, 1.900390625, -0.9013671875, -0.6435546875, -1.05859375, -0.01486968994140625, 0.5693359375, 1.912109375, 0.9794921875, -0.311279296875, -1.37109375, -0.39990234375, -1.2021484375, -0.412353515625, 1.1279296875, -0.61328125, 1.201171875, -1.7509765625, -0.83544921875, -0.037017822265625, 1.0439453125, 1.6298828125, -0.28857421875, -2.560546875, -1.0322265625, -0.212646484375, -0.2169189453125, 1.5791015625, -1.505859375, -0.491943359375, -0.043609619140625, 0.6142578125, 0.95751953125, 1.5205078125, 1.8046875, -0.153564453125, 2.279296875, 0.21484375, -0.53857421875, -1.1630859375, -0.431884765625, -0.2548828125, -0.2178955078125, 1.0107421875, 0.685546875, 0.712890625, 1.533203125, 0.18408203125, -0.88134765625, -1.099609375, -0.8515625, 1.5498046875, -0.7109375, 0.8623046875, -0.353271484375, -0.1556396484375, -0.9189453125, -0.90478515625, -2.107421875, -0.187255859375, -1.681640625, 0.487060546875, 0.93896484375, 0.22900390625, 0.265380859375, -0.68408203125, 0.354736328125, 1.9873046875, 0.5908203125, 0.137939453125, 0.09405517578125, 0.5400390625, 0.4912109375, -1.1005859375, 0.55224609375, 0.33544921875, -0.556640625, 0.2203369140625, -0.72314453125, 0.2467041015625, -1.3837890625, 0.1085205078125, 0.441162109375, 0.490478515625, -0.3466796875, 0.640625, 1.294921875, -1.0068359375, -0.2099609375, -2.125, 0.262451171875, 0.55419921875, 0.84033203125, -0.420654296875, -0.1728515625, 0.77978515625, 0.48291015625, -0.242431640625, 0.70703125, 0.39404296875, 0.8994140625, 0.6416015625, -0.156982421875, -0.80908203125, -0.5517578125, 1.4814453125, -0.630859375, 0.986328125, -1.177734375, -0.059234619140625, -1.5966796875, -2.037109375, -0.338134765625, 0.86181640625, -0.0491943359375, -1.55078125, 0.158935546875, 0.0028934478759765625, 1.2705078125, -1.404296875, -0.2381591796875, -1.154296875, -0.8486328125, 0.82958984375, 1.208984375, -0.87744140625, -2.017578125, -1.0556640625, 0.82666015625, 1.15625, 1.0830078125, 0.60546875, -0.12347412109375, 1.7509765625, 0.87744140625, 0.273193359375, 0.034088134765625, -1.1494140625, -2.669921875, 1.08984375, 0.5888671875, -1.755859375, -0.2802734375, -1.609375, 0.9931640625, 0.2353515625, 1.673828125, -0.0016031265258789062, -1.6533203125, -0.412109375, 0.2115478515625, -1.7578125, 0.436767578125, -0.52294921875, 1.34375, -1.4580078125, -0.90673828125, -0.70166015625, 0.486083984375, 0.8544921875, 0.79345703125, 0.6943359375, 1.0166015625, -0.390380859375, -0.68798828125, 0.3271484375, -0.11505126953125, 1.4130859375, -0.301025390625, -0.77001953125, -0.787109375, -0.317626953125, 1.57421875, 0.5439453125, -1.6044921875, -0.53515625, -0.275634765625, 0.64794921875, 1.2255859375, 1.44921875, 1.1982421875, 1.7001953125, 0.73388671875, -0.1649169921875, 1.0322265625, -1.359375, 0.859375, -0.39501953125, -0.580078125, -1.771484375, 0.16943359375, 0.61962890625, -0.9013671875, 1.0693359375, 0.7705078125, -1.50390625, -0.408935546875, -0.397705078125, -0.80517578125, 1.349609375, 0.355224609375, -1.0673828125, 2.1015625, 0.1343994140625, 0.41650390625, 0.2080078125, -0.5087890625, -0.67138671875, -0.27734375, -0.5810546875, 0.425048828125, 1.4384765625, -0.5107421875, 2.162109375, -1.515625, -0.2086181640625, -0.423095703125, 1.4306640625, 1.70703125, 0.0238494873046875, -0.40869140625, -3.67578125, 1.287109375, -1.892578125, 0.162109375, -0.78125, -0.261474609375, 1.373046875, 1.06640625, 1.12890625, -0.69287109375, -1.6533203125, 0.1942138671875, 0.66748046875, 0.666015625, -0.94677734375, 1.0771484375, 0.5830078125, -0.2998046875, 1.0625, -1.48828125, -1.630859375, 0.38037109375, 0.81689453125, -1.58984375, -1.25, -0.79541015625, -1.3359375, 0.423828125, 1.0283203125, -0.343505859375, -0.04949951171875, 0.0155792236328125, 1.05078125, 1.91796875, -0.5888671875, 0.328125, -0.6298828125, -1.1337890625, 0.71435546875, -0.23486328125, -2.162109375, 0.62548828125, 0.299072265625, 0.005985260009765625, 0.27294921875, -0.37451171875, 1.2041015625, 0.37451171875, -1.1728515625, -0.42138671875, -0.94384765625, -0.30810546875, 0.64892578125, 1.4423828125, 0.178955078125, 0.320556640625, -0.7373046875, -1.3203125, -0.2044677734375, -1.2216796875, 1.2587890625, 0.08160400390625, 0.8212890625, -0.64404296875, 0.86181640625, 0.66748046875, 0.381103515625, -0.27099609375, -0.39599609375, -1.2900390625, -0.33056640625, 0.56591796875, 0.313720703125, -0.5693359375, -0.392333984375, 0.744140625, -1.673828125, 0.0203857421875, -1.5546875, -3.404296875, 0.234619140625, -1.35546875, 1.3037109375, -0.09039306640625, 0.1923828125, -0.62841796875, 0.492919921875, -0.7275390625, 1.9189453125, -0.5146484375, 2.09375, -0.41064453125, 0.955078125, -0.0894775390625, 0.320068359375, -1.6171875, -1.3017578125, -1.0869140625, 0.86962890625, -1.3330078125, -1.4208984375, -1.1513671875, 1.5654296875, -2.083984375, -0.50439453125, 1.09765625, -0.375732421875, -0.11785888671875, 0.73486328125, 0.30419921875, 0.3330078125, -2.017578125, 0.42822265625, 1.3876953125, 0.5654296875, 1.03515625, -1.2890625, -0.2939453125, 0.38720703125, -0.66650390625, -0.40576171875, 0.1478271484375, -1.1279296875, 0.76416015625, -0.338134765625, -0.44091796875, 0.120849609375, 0.414794921875, -1.037109375, 1.728515625, -2.25, 1.6787109375, -0.8046875, -1.578125, -0.58056640625, -0.10601806640625, -0.1781005859375, -0.289306640625, -1.7265625, -0.041534423828125, -0.11480712890625, -0.7529296875, -0.287109375, 1.9306640625, -1.4609375, -0.80517578125, -1.9453125, -1.4345703125, -0.35205078125, -1.0947265625, -0.1767578125, 0.01190948486328125, -0.8447265625, 1.0791015625, 0.227294921875, -0.48974609375, 0.73388671875, 0.892578125, -0.72705078125, 0.1744384765625, 0.350341796875, -0.5625, -1.759765625, -1.2158203125, -0.220947265625, -0.1676025390625, -0.279052734375, -2.083984375, -1.3828125, -0.1561279296875, 0.27099609375, 1.5380859375, 0.638671875, 1.62109375, -0.9267578125, -0.47900390625, 2.365234375, -0.39892578125, -1.9892578125, 0.93408203125, 0.2437744140625, -0.71728515625, 1.0380859375, -1.1025390625, -0.96826171875, 1.5517578125, -0.310791015625, -1.9384765625, 1.609375, -1.2919921875, 1.099609375, -1.033203125, -0.1915283203125, 0.126220703125, -0.09619140625, -0.654296875, -0.09906005859375, -0.06793212890625, -2.0625, 0.4375, -0.98388671875, 1.162109375, -0.5078125, 1.70703125, -0.437255859375, 0.2088623046875, -2.240234375, -0.48095703125, 0.7998046875, 1.1025390625, -1.3837890625, -0.80859375, -1.0107421875, 1.0986328125, 0.80419921875, -1.7490234375, -2.06640625, -0.467041015625, -1.6337890625, 1.4580078125, -0.53173828125, -1.2783203125, -0.6337890625, -0.274658203125, -0.336181640625, 0.61181640625, 0.93408203125, 1.2373046875, -0.6806640625, 0.48681640625, -0.402099609375, -1.6923828125, -0.427001953125, -0.1478271484375, -2.494140625, -0.6845703125, -0.330810546875, 0.7138671875, -1.5830078125, 1.478515625, -1.7109375, 0.1849365234375, -0.008697509765625, -1.9326171875, -0.8486328125, 0.428466796875, 0.80712890625, 0.1192626953125, 0.4052734375, 0.06768798828125, 1.6455078125, -0.78515625, -0.7783203125, 0.2359619140625, 1.10546875, -1.400390625, 0.6376953125, 0.1485595703125, 1.162109375, -1.2548828125, 1.4072265625, 1.8662109375, 1.10546875, -0.41943359375, 0.65771484375, -0.2998046875, 0.1839599609375, 1.0615234375, 1.0830078125, -0.59033203125, 1.142578125, -1.3154296875, -0.036865234375, 1.46484375, 0.412353515625, 1.4345703125, -0.97216796875, 1.8359375, 1.0859375, -0.137939453125, -2.697265625, -0.87060546875, -0.461181640625, -0.144287109375, -0.77099609375, 1.1240234375, 0.86279296875, -0.66943359375, 0.474365234375, 0.138671875, 0.2880859375, -0.247802734375, -0.24267578125, -1.111328125, 0.04888916015625, 0.54345703125, 0.947265625, 0.92529296875, 1.02734375, -0.289794921875, 0.6689453125, -0.5859375, -0.3720703125, -0.55712890625, -0.6611328125, 1.8857421875, -0.55029296875, 0.2442626953125, 0.77685546875, 1.69921875, 0.330322265625, -1.025390625, 0.2054443359375, 2.765625, -0.391845703125, 0.9609375, 1.71875, -0.70751953125, -0.56005859375, 0.720703125, -0.67431640625, -1.18359375, -0.2186279296875, -0.290283203125, -0.7421875, 0.923828125, -0.2177734375, -0.2203369140625, 1.33984375, 0.93408203125, -1.2236328125, -0.326904296875, 0.78515625, 1.484375, -0.40576171875, 1.521484375, -0.580078125, 0.3759765625, 0.99267578125, 0.830078125, 1.177734375, 1.01171875, 0.052581787109375, 0.11395263671875, 0.1484375, 1.1826171875, 0.853515625, -1.7412109375, 0.8837890625, 0.411865234375, -1.5126953125, 1.3955078125, 0.595703125, -0.0972900390625, -0.435546875, -0.1695556640625, -0.254638671875, 0.943359375, -0.453857421875, -0.448486328125, 1.0078125, 1.177734375, 0.371337890625, -1.3935546875, 0.248291015625, -0.291259765625, -0.96484375, 0.97314453125, -0.17724609375, 0.9892578125, -0.93017578125, -0.55126953125, -1.353515625, -0.5751953125, -0.437744140625, 0.1024169921875, 1.7490234375, -1.6982421875, 0.125244140625, 0.416015625, -0.6826171875, 0.521484375, -0.93701171875, -0.12396240234375, 0.91650390625, 0.080322265625, 0.611328125, 2.64453125, -0.397705078125, -0.1873779296875, -0.50244140625, -0.37939453125, -1.4150390625, 0.74560546875, 0.970703125, 0.0221405029296875, 1.720703125, 0.029022216796875, 0.87353515625, -0.26806640625, -1.64453125, -1.6181640625, 0.92041015625, 0.437255859375, -0.3662109375, -1.544921875, -0.8798828125, 0.7802734375, -0.2469482421875, 0.332275390625, 0.1778564453125, -0.57958984375, 0.0150909423828125, -1.224609375, -1.1904296875, 0.587890625, 0.080078125, 0.66552734375, 1.146484375, -0.07879638671875, -0.457275390625, 0.334716796875, 0.264404296875, -0.63330078125, 0.79052734375, -0.95751953125, -0.394775390625, -0.10687255859375, -1.32421875, 1.24609375, -0.16162109375, -0.49755859375, -0.25732421875, -0.4677734375, -0.318115234375, 0.45458984375, 0.7373046875, 1.525390625, -0.350830078125, 1.8125, -0.74951171875, 0.5556640625, 1.1982421875, -0.33251953125, -0.1165771484375, 0.61865234375, -0.22314453125, 0.4990234375, -0.4013671875, -0.359130859375, 0.5869140625, 2.048828125, 2.716796875, 1.27734375, -1.6943359375, -1.9443359375, -0.68701171875, 0.75927734375, 1.2666015625, -0.2724609375, -0.73046875, 0.59033203125, -0.41943359375, -0.7021484375, -0.0199432373046875, 0.270263671875, 1.1953125, 1.3486328125, 0.2491455078125, -1.384765625, 1.9794921875, -0.78857421875, -1.1513671875, 1.0205078125, 1.03515625, -1.40234375, -0.8447265625, 0.7685546875, 0.0306243896484375, 0.0361328125, 0.32421875, -0.3896484375, 0.120361328125, -0.93505859375, 0.93896484375, -0.2359619140625, -0.2763671875, 1.41796875, 0.263916015625, 0.195068359375, 1.2177734375, -2.0234375, -1.6875, 1.6416015625, 0.332763671875, -0.1444091796875, 0.763671875, 0.8125, -0.87255859375, -1.5498046875, 0.1580810546875, -0.86669921875, 0.93896484375, 1.7099609375, -1.4306640625, -1.0087890625, 0.44677734375, 0.759765625, 0.00542449951171875, -0.05010986328125, -0.58056640625, 0.1707763671875, 0.77001953125, -2.080078125, -1.28125, -0.366455078125, -0.29248046875, -0.033782958984375, 0.8056640625, -0.6689453125, 0.97509765625, 1.9140625, -0.335693359375, -0.345458984375, 0.0496826171875, 0.0665283203125, -1.62890625, -0.302001953125, 0.26953125, 1.85546875, 0.047821044921875, -1.0908203125, 0.138427734375, 0.187255859375, -0.00878143310546875, 1.2119140625, -0.474609375, 1.734375, 0.8193359375, -0.425048828125, 0.60986328125, 0.9267578125, -0.61279296875, 1.146484375, -0.87890625, -1.9794921875, -0.31005859375, -0.74609375, 0.97802734375, -1.912109375, 0.712890625, 1.5166015625, -0.9599609375, -0.2220458984375, 1.21875, 0.93896484375, 1.10546875, -1.501953125, 1.8369140625, 0.75537109375, 0.485107421875, -0.123046875, -0.0631103515625, -1.349609375, 0.054840087890625, 0.61181640625, -0.048065185546875, -1.8525390625, -1.5986328125, 1.8076171875, 1.6552734375, -0.3564453125, 1.62109375, 0.59033203125, -2.513671875, -0.5283203125, 0.9052734375, 0.935546875, 1.939453125, 0.020599365234375, -0.1341552734375, -0.195068359375, 1.7529296875, 0.7734375, 0.8427734375, -0.354736328125, 0.65625, 0.5595703125, -0.2127685546875, -1.54296875, -0.142578125, 0.295166015625, 0.48486328125, 0.01023101806640625, -0.95556640625, 1.134765625, 0.62060546875, 0.6728515625, -0.2724609375, -0.60400390625, -0.8798828125, -1.1279296875, 0.45068359375, -0.11846923828125, -0.12548828125, 1.1376953125, 0.2265625, -0.873046875, 0.65673828125, 0.3486328125, -0.78857421875, 0.0235443115234375, 0.8759765625, 0.126708984375, 0.440185546875, 0.5205078125, -1.251953125, -1.6259765625, 1.5390625, -0.1160888671875, -2.0703125, -0.80859375, -0.1458740234375, 0.94921875, 0.41455078125, -1.498046875, -2.083984375, -0.063232421875, -2.01171875, -0.36767578125, -0.1495361328125, -0.39990234375, 1.28515625, 0.251953125, 2.302734375, -0.49951171875, 0.2215576171875, 2.462890625, -0.337890625, 1.2587890625, -2.35546875, -0.50146484375, 0.230224609375, -0.351318359375, -0.56103515625, 0.56640625, 0.07763671875, -1.8896484375, -0.280517578125, -0.673828125, -0.426025390625, 1.4990234375, -1.6787109375, 1.865234375, -0.3525390625, 0.80712890625, -1.1611328125, 0.359130859375, 0.260986328125, 1.1689453125, 0.61328125, 1.5771484375, 0.12127685546875, 0.5341796875, -0.59912109375, -0.43408203125, -0.0869140625, 1.5849609375, 1.0234375, -2.625, 1.80078125, 1.998046875, 0.52294921875, -0.73779296875, 0.30615234375, 0.6865234375, -0.31591796875, -1.08984375, -1.236328125, -0.0762939453125, 1.63671875, 2.46875, 0.43359375, 0.2451171875, 0.482421875, 0.0245361328125, -0.21533203125, -0.221923828125, -0.126708984375, -0.4638671875, 1.484375, -0.578125, 0.29248046875, 0.33544921875, 0.01111602783203125, 0.1375732421875, -1.2353515625, 0.1590576171875, -1.23828125, -0.334716796875, 0.88427734375, 1.11328125, 0.677734375, 1.650390625, 0.09100341796875, -0.72705078125, 0.033355712890625, 0.7353515625, 0.1483154296875, 1.533203125, -1.5654296875, -0.689453125, -1.3857421875, 0.9638671875, -1.03515625, -0.84033203125, -0.11737060546875, 1.7880859375, 1.072265625, 0.375, 0.83740234375, 1.4453125, -0.77099609375, 1.546875, 0.10626220703125, 1.3896484375, 2.111328125, -1.2861328125, -0.45068359375, 0.04949951171875, -1.2392578125, 0.339599609375, -1.763671875, 0.154052734375, 0.6435546875, 0.57373046875, 0.92431640625, 1.138671875, -0.7470703125, -0.7724609375, -1.58984375, -0.052825927734375, -1.0771484375, -0.08160400390625, 0.81884765625, -0.1595458984375, 0.4443359375, 0.2568359375, 0.004924774169921875, -0.032318115234375, 1.46484375, 0.160888671875, 0.195556640625, -0.2705078125, -0.91357421875, 1.076171875, -0.63427734375, 0.4091796875, 0.08355712890625, -0.043060302734375, 0.34033203125, -0.410400390625, -0.90283203125, -1.2958984375, 0.1363525390625, 0.7958984375, 0.2374267578125, 0.280517578125, -2.046875, 0.83251953125, -0.20751953125, -0.09307861328125, -1.3876953125, 1.2177734375, -1.087890625, 0.63916015625, -0.316650390625, -0.045135498046875, -0.2125244140625, -1.17578125, 0.08489990234375, -0.8525390625, 0.351318359375, -1.4833984375, -0.75830078125, 0.386962890625, -1.205078125, -1.166015625, -0.7890625, -0.6904296875, -0.37841796875, -0.75634765625, 1.111328125, 0.24169921875, -0.5791015625, -0.135498046875, -1.3134765625, -0.2344970703125, 1.75390625, -0.7490234375, 0.68359375, 0.96875, -1.0712890625, 0.17626953125, -0.41748046875, 0.65625, 1.65234375, 0.50830078125, -1.3525390625, -1.208984375, 1.2060546875, -0.9443359375, -0.1588134765625, -0.1107177734375, 1.4453125, 0.3134765625, 0.45068359375, -0.315673828125, 0.439453125, 0.1510009765625, -0.1802978515625, -1.0458984375, -1.146484375, 0.266357421875, -0.53955078125, -0.6953125, -2.322265625, 1.958984375, -1.0498046875, 0.513671875, -0.69921875, 0.378173828125, -0.3193359375, -0.9560546875, 1.1669921875, 0.86279296875, -0.65576171875, -0.990234375, -0.34326171875, 0.70849609375, -0.365478515625, 0.0003247261047363281, 1.01953125, -1.748046875, 1.453125, -1.642578125, 0.720703125, -0.150634765625, -0.36865234375, 0.84521484375, -0.09637451171875, -0.36669921875, -1.2802734375, 0.685546875, -0.068603515625, -0.998046875, 1.779296875, 0.5810546875, 0.2327880859375, 2.23828125, -1.6044921875, -0.52490234375, 0.71240234375, 0.46923828125, -1.30859375, -1.1494140625, 0.345703125, 0.0259857177734375, 0.4814453125, -1.4365234375, -0.3173828125, 0.73876953125, 0.564453125, -0.75048828125, 0.294677734375, 0.414306640625, -0.47314453125, 0.0633544921875, 2.580078125, -0.603515625, 0.29248046875, 0.814453125, -1.02734375, -0.9794921875, 1.4130859375, 0.369873046875, 0.30810546875, 1.203125, -1.3349609375, -0.07080078125, 0.6171875, 0.697265625, 0.123046875, 1.12109375, -0.85986328125, 1.0576171875, -0.156982421875, -0.7177734375, 0.289794921875, 0.447509765625, -1.947265625, -0.98095703125, -0.148193359375, 0.372802734375, 0.2420654296875, -0.82763671875, 1.220703125, 0.501953125, -0.333251953125, 0.22998046875, -0.798828125, -0.20556640625, 0.900390625, 1.8916015625, 1.52734375, -1.3076171875, 2.087890625, 0.70166015625, -1.7939453125, -1.224609375, 0.6240234375, 1.2431640625, -0.370361328125, -0.022857666015625, 0.81689453125, 1.275390625, -0.6962890625, 1.16796875, 1.580078125, -0.152587890625, -0.9404296875, -0.057586669921875, -0.7900390625, -3.484375, 0.615234375, 0.1021728515625, 0.91943359375, 1.0126953125, -0.4658203125, 1.9140625, -1.1572265625, 1.1865234375, -1.2685546875, -1.232421875, -0.2222900390625, -0.796875, 0.80419921875, 0.24267578125, 1.0791015625, -0.69677734375, -0.319091796875, -0.1317138671875, -0.016326904296875, 0.96484375, 1.1845703125, 0.51953125, -1.482421875, -0.96484375, 0.80810546875, -0.35986328125, 0.69580078125, 0.7216796875, -0.3251953125, 0.434814453125, 1.806640625, 0.73193359375, 1.1142578125, -1.4375, 2.294921875, 1.0341796875, -1.4140625, -0.377685546875, 0.58251953125, -0.0411376953125, -0.157958984375, 0.0242462158203125, -1.1591796875, -0.76708984375, 0.16259765625, -1.359375, 0.95263671875, -0.28515625, -0.11566162109375, -0.466064453125, -1.2197265625, 0.04803466796875, 0.6005859375, -0.3447265625, -0.8046875, -0.437744140625, 1.1611328125, -0.52099609375, -0.66162109375, 0.320068359375, 0.07958984375, 0.990234375, -0.2340087890625, -1.16015625, -0.95361328125, 0.1363525390625, -1.0048828125, -0.22998046875, -1.8876953125, 1.083984375, -0.47314453125, 0.7958984375, 0.53662109375, -0.189697265625, -0.7783203125, 0.587890625, 0.49951171875, -0.7841796875, 0.429443359375, -0.7119140625, 0.148681640625, -0.95361328125, -0.68701171875, 2.189453125, -0.472412109375, 0.00704193115234375, 0.10504150390625, 2.169921875, -0.1136474609375, 1.4814453125, 0.22119140625, 0.79443359375, -1.0009765625, -1.74609375, 0.7880859375, -0.1602783203125, -0.63134765625, 1.5205078125, -0.037017822265625, 1.4208984375, 0.18115234375, 1.2451171875, -0.9384765625, -0.1767578125, 0.2352294921875, -0.361083984375, 0.42529296875, 2.126953125, -0.341552734375, -0.1380615234375, 0.177734375, -1.6796875, 1.25390625, 0.2900390625, 0.56103515625, 1.3056640625, -1.9892578125, -0.275146484375, 0.833984375, 0.9580078125, 0.3818359375, 0.037078857421875, -0.85595703125, 0.26220703125, -0.1846923828125, -0.281982421875, -1.44140625, 1.2099609375, 0.2822265625, 0.6083984375, -1.7216796875, -0.552734375, -0.058441162109375, 1.1748046875, 0.22314453125, 0.5185546875, -0.54052734375, 0.136474609375, -1.767578125, -0.72314453125, 2.05859375, 0.64111328125, -1.5341796875, -0.50537109375, 0.8310546875, 0.9248046875, -0.86279296875, -0.470703125, 0.72412109375, 0.388427734375, 1.2451171875, 0.36328125, 0.28271484375, 1.1123046875, -0.10162353515625, -0.33154296875, -1.1474609375, 0.93896484375, 1.548828125, -0.40234375, -0.287841796875, -1.701171875, 1.568359375, -1.3154296875, -1.65234375, 2.08984375, 1.822265625, -0.74267578125, 0.09637451171875, 0.00046634674072265625, -1.728515625, 0.416259765625, -0.1390380859375, 1.580078125, 1.3427734375, 1.583984375, 1.947265625, 0.5244140625, -0.3193359375, 0.54638671875, -0.5869140625, 0.86376953125, -1.529296875, -1.1376953125, 0.59716796875, -0.07012939453125, 0.423095703125, -1.2509765625, 0.7978515625, -2.017578125, -0.1385498046875, 0.409423828125, -0.1136474609375, -0.399658203125, 0.1712646484375, -0.5078125, 0.82763671875, 1.7666015625, -0.32421875, -0.1336669921875, -0.8583984375, -0.4912109375, 1.32421875, 0.232421875, -1.5859375, 0.60400390625, 0.71923828125, -1.1669921875, 0.0181732177734375, 1.2060546875, 1.021484375, 1.48828125, 1.42578125, 0.6904296875, 1.6787109375, -0.7822265625, 0.8701171875, 1.2890625, -1.4755859375, -0.25732421875, 1.7822265625, 0.2120361328125, 0.278076171875, -1.369140625, -0.5419921875, -0.58740234375, 1.0849609375, -1.1259765625, 1.3525390625, 0.744140625, -1.400390625, -1.2626953125, 1.0546875, -1.0283203125, 0.8154296875, -1.5078125, 0.56396484375, 0.006755828857421875, -1.7900390625, -0.76708984375, 1.28125, 1.982421875, -1.642578125, 0.275390625, -0.40380859375, -0.07781982421875, 0.313720703125, -0.5478515625, -0.73291015625, -0.65380859375, -2.021484375, 0.239990234375, -0.978515625, 0.54736328125, 1.8212890625, -1.044921875, 0.0243988037109375, 0.398681640625, 0.448974609375, 2.009765625, 0.84228515625, -0.89404296875, 0.9853515625, -0.64453125, -1.787109375, 0.12298583984375, 1.1298828125, 0.1859130859375, 0.89453125, 0.024322509765625, 2.48828125, 1.0634765625, -0.7080078125, -1.037109375, 1.7666015625, 0.367431640625, -0.6435546875, -1.1552734375, 1.5009765625, 0.6513671875, 0.2110595703125, -1.59765625, -0.032470703125, -0.3896484375, 0.31298828125, 1.3642578125, -0.6201171875, 0.5888671875, 0.30615234375, 0.0726318359375, 0.712890625, -1.169921875, 1.279296875, 0.6494140625, 0.372314453125, -0.58935546875, -0.92431640625, 0.057952880859375, -0.480224609375, 0.52734375, -0.446044921875, -4.0703125, -1.3037109375, -0.359130859375, 1.001953125, -0.42236328125, 0.66748046875, 1.431640625, 0.34228515625, 1.2626953125, 1.0126953125, -0.15625, -0.478759765625, 0.026123046875, 0.5224609375, 0.80078125, 1.3330078125, 0.417236328125, -1.689453125, -0.133056640625, -0.55078125, -1.7255859375, 1.171875, 0.92529296875, 2.244140625, 0.72705078125, -0.446044921875, -1.8154296875, 1.26171875, -0.98388671875, -0.921875, -2.154296875, -1.029296875, 0.5205078125, -1.1494140625, 1.6943359375, 2.271484375, 0.254638671875, 1.4541015625, -1.4970703125, 0.91015625, 0.8994140625, -0.05230712890625, -0.301025390625, -0.0941162109375, 0.1785888671875, 0.1336669921875, 0.38525390625, 1.259765625, -0.82958984375, -0.296875, -1.8125, -0.38427734375, -0.92626953125, -1.25, -0.91015625, -0.491455078125, -0.8505859375, 0.70751953125, 0.0916748046875, 0.90087890625, 0.46533203125, 1.4423828125, 0.373046875, 1.416015625, 0.59619140625, -2.083984375, -1.1083984375, -1.6298828125, 0.313720703125, 0.1500244140625, -0.115478515625, -0.9541015625, 1.220703125, -1.6650390625, 0.403076171875, -1.0693359375, 0.0031566619873046875, 1.2509765625, 0.32666015625, -0.90380859375, 0.79345703125, 1.8115234375, -0.92626953125, 0.4912109375, -0.8251953125, 0.36865234375, -0.70751953125, -1.1171875, 0.8193359375, -0.381103515625, 0.75634765625, -0.0986328125, 0.35595703125, -0.72802734375, -0.198974609375, 1.2978515625, 0.38623046875, 0.041900634765625, -0.712890625, -1.59375, -1.076171875, -0.7431640625, -1.462890625, 0.54638671875, 0.62548828125, 1.41796875, 1.2021484375, -0.9423828125, -0.151123046875, 1.1435546875, -0.2099609375, -0.2288818359375, -0.10931396484375, 1.0009765625, 0.9873046875, -1.2822265625, 0.51806640625, 0.96728515625, 0.5400390625, -0.1468505859375, -0.1085205078125, -0.8310546875, 1.3623046875, -2.009765625, 1.3984375, -0.80810546875, -0.6953125, -0.364013671875, 0.218017578125, 0.5595703125, -0.2027587890625, -1.7333984375, 0.49658203125, 1.5078125, 1.0595703125, 0.3359375, 0.419921875, 0.97021484375, 0.20751953125, 0.71337890625, 0.04461669921875, -0.2132568359375, 0.689453125, 0.0966796875, 0.26123046875, 0.333251953125, -1.49609375, 1.5302734375, 0.10772705078125, 1.6923828125, -0.137939453125, 0.60498046875, 0.213623046875, 0.1322021484375, 2.330078125, -0.93115234375, -2.03515625, 1.236328125, -1.384765625, 0.791015625, -1.7265625, -0.51123046875, 0.2427978515625, -0.2281494140625, -1.31640625, 0.57275390625, 0.037872314453125, 0.366455078125, -0.3447265625, 1.1904296875, -1.046875, -0.666015625, 1.2861328125, -0.40966796875, 1.623046875, 1.162109375, 0.330322265625, -0.042236328125, 0.7255859375, 0.8701171875, 0.2371826171875, -0.60400390625, 0.90185546875, 0.82666015625, -1.14453125, 0.1773681640625, 1.2138671875, 0.3271484375, 1.0966796875, 0.4072265625, -0.67724609375, 0.15087890625, -0.60400390625, -0.68310546875, 0.9521484375, -1.1640625, -1.1572265625, -2.583984375, -0.27587890625, 0.70556640625, -0.951171875, -0.53564453125, -1.4130859375, 0.5517578125, -0.2939453125, -0.2017822265625, 0.07861328125, -1.6708984375, -0.50048828125, 0.1986083984375, 0.134521484375, -0.5478515625, 1.7392578125, 1.6376953125, -0.1014404296875, 0.473388671875, -0.5673828125, 0.3193359375, 0.278076171875, -2.36328125, -1.15234375, -0.1973876953125, 0.53662109375, 0.63525390625, 0.09796142578125, 0.7607421875, 0.359619140625, 0.33251953125, 1.1220703125, 0.28466796875, -0.1864013671875, 0.91259765625, -1.05859375, -0.71484375, 0.251708984375, 0.151123046875, -0.1575927734375, -1.7041015625, 0.00362396240234375, 0.72265625, -1.33203125, -0.634765625, -0.79736328125, -0.0084075927734375, -0.74072265625, -0.2061767578125, 1.4736328125, 0.338623046875, 0.560546875, -0.8896484375, -1.3232421875, -0.2430419921875, -0.59912109375, 0.14208984375, -0.44189453125, -1.01171875, 1.0537109375, -1.1650390625, 0.138427734375, -1.5966796875, 0.81005859375, -2.2734375, -0.2369384765625, -0.1658935546875, 1.19140625, 0.319091796875, -0.31298828125, -1.1796875, 0.97705078125, -0.336669921875, 1.5361328125, -1.5498046875, 1.830078125, -1.654296875, -0.047119140625, -0.8134765625, -0.046295166015625, 1.9921875, 0.046478271484375, -2.013671875, 0.2078857421875, -0.91015625, 0.763671875, 0.2919921875, -0.464599609375, -0.350341796875, 1.705078125, -1.849609375, 0.9384765625, -0.006855010986328125, -0.0032558441162109375, -0.818359375, -1.3994140625, 0.0138092041015625, 0.08612060546875, -0.349365234375, 1.0625, -1.1611328125, 0.759765625, -1.6357421875, -0.12939453125, 1.4150390625, 2.232421875, 0.91943359375, -1.962890625, -0.80224609375, 0.11138916015625, 0.6279296875, -0.30712890625, -0.98828125, 0.1422119140625, 1.1689453125, -0.900390625, -1.3662109375, -0.58544921875, -0.994140625, -0.7216796875, -1.3701171875, -0.361328125, -0.5302734375, -0.75146484375, 1.1904296875, 0.6591796875, -1.041015625, 1.7978515625, -0.09039306640625, 0.01132965087890625, 1.015625, -1.130859375, -1.859375, 0.02947998046875, 0.5341796875, -0.7275390625, 0.74462890625, -0.086669921875, 0.54931640625, 0.5458984375, 0.1114501953125, -1.990234375, 0.97509765625, -1.5751953125, -0.529296875, 0.19921875, -0.0012445449829101562, -0.0938720703125, -2.337890625, 1.033203125, -0.08013916015625, 0.87744140625, 0.041290283203125, 0.265380859375, 1.6865234375, 0.3662109375, 0.83935546875, -0.313232421875, -0.2484130859375, 1.3837890625, -0.11083984375, 1.5263671875, -0.93115234375, 0.8154296875, -0.2110595703125, 0.88525390625, 0.283447265625, -0.66650390625, 0.15869140625, -1.08203125, 0.365966796875, -1.5380859375, 0.11578369140625, -0.258544921875, 0.403076171875, 0.346435546875, -2.43359375, 1.41796875, 0.68994140625, 0.568359375, 1.7626953125, -0.400634765625, 1.92578125, -0.3486328125, -1.8154296875, 0.55859375, -1.125, 0.6572265625, 0.4951171875, -0.367431640625, 0.16943359375, 1.6787109375, 0.8974609375, 1.0185546875, 0.374267578125, 0.28662109375, 0.302001953125, 0.343994140625, -0.052825927734375, -1.248046875, 0.145751953125, 0.3115234375, 0.91650390625, 0.10198974609375, -1.3681640625, -0.292236328125, -0.1375732421875, 0.52197265625, -2.333984375, -0.595703125, 0.1304931640625, -0.495849609375, 0.65625, 0.86767578125, -2.29296875, -0.75390625, -0.00560760498046875, 0.386962890625, 2.966796875, -0.97900390625, 1.5400390625, 1.06640625, -0.73974609375, -0.1617431640625, -1.52734375, -0.48046875, 0.77001953125, -0.00405120849609375, -0.177734375, -0.056396484375, -1.5166015625, 1.091796875, 0.345703125, 0.39404296875, -1.748046875, 1.1064453125, -1.1630859375, 0.08465576171875, -1.8671875, -1.9462890625, -1.2236328125, -0.60205078125, -0.1552734375, -0.71240234375, -0.156494140625, -2.326171875, -0.020233154296875, -1.05859375, -0.876953125, 0.73779296875, 0.8564453125, -1.0830078125, -0.65380859375, 0.1922607421875, 1.587890625, 0.42724609375, -0.399169921875, -0.501953125, 0.63427734375, -0.136962890625, -0.3408203125, -0.2568359375, -0.57421875, 0.88037109375, 0.25341796875, -0.1676025390625, -1.166015625, 1.5322265625, -0.02178955078125, -0.56640625, 0.5234375, -0.12078857421875, -1.98828125, -1.5166015625, -0.07464599609375, 0.425048828125, 0.9697265625, -0.94140625, 1.486328125, 0.61376953125, -0.102294921875, -0.828125, 0.68701171875, -0.166015625, 1.0498046875, 0.0038166046142578125, -0.11016845703125, 0.173095703125, -0.99365234375, 0.319580078125, 0.403564453125, 1.451171875, -1.8701171875, 1.103515625, -0.78466796875, 0.515625, 0.045318603515625, 0.75927734375, -1.494140625, -0.28662109375, 0.1864013671875, 0.479248046875, 0.50048828125, 0.12646484375, 1.216796875, 0.61865234375, -0.370849609375, 0.81591796875, 0.1834716796875, -0.619140625, 0.77197265625, -1.8935546875, 0.81689453125, 1.263671875, 1.3505859375, 2.130859375, 1.5166015625, -1.2412109375, -0.63671875, -0.06024169921875, -0.08624267578125, -0.95849609375, -0.94580078125, 0.252685546875, 0.040130615234375, -1.4169921875, 0.1324462890625, 1.08203125, 1.888671875, -0.77294921875, -0.305908203125, -1.314453125, 1.1005859375, -0.143310546875, 1.572265625, 0.7861328125, 0.402587890625, 0.72412109375, 0.36279296875, -0.0389404296875, 1.0859375, 0.625, -1.212890625, 1.34375, -1.509765625, -0.27880859375, -1.0068359375, -1.0390625, 0.9736328125, -1.140625, -0.78125, 0.6435546875, -0.332275390625, 0.311767578125, -0.2174072265625, 0.1234130859375, 0.20166015625, 1.4990234375, 0.00977325439453125, -1.7734375, -0.042938232421875, 0.0155792236328125, 1.21484375, -0.54931640625, 0.1044921875, 0.13916015625, 1.0556640625, -1.2783203125, 0.978515625, -0.65966796875, -0.744140625, -0.693359375, 0.5087890625, -0.130615234375, 0.98583984375, -0.156005859375, -1.0830078125, -1.2001953125, 0.876953125, 1.15234375, -0.7509765625, 0.0660400390625, -0.64111328125, -0.62451171875, -0.447021484375, -0.149658203125, -1.099609375, -1.6728515625, 0.81103515625, 1.1650390625, -0.5361328125, 0.01038360595703125, -1.125, -1.3447265625, -1.2451171875, 1.8125, -0.59228515625, -0.43701171875, -0.77783203125, 0.5537109375, -0.439453125, 0.406494140625, 0.859375, -1.765625, -1.90234375, -0.0303802490234375, -1.3720703125, -1.6767578125, 0.10198974609375, -1.3154296875, -0.61474609375, 0.666015625, 0.2213134765625, -1.7978515625, -0.1619873046875, 0.384521484375, -0.0999755859375, -0.058502197265625, -2.037109375, 1.087890625, -1.3251953125, -0.27294921875, 0.1708984375, 0.5205078125, -0.59765625, 0.85498046875, 0.9951171875, 1.1630859375, 1.4638671875, -0.0206298828125, -0.49609375, -1.021484375, -1.2158203125, -0.6162109375, -0.07403564453125, -1.376953125, 1.11328125, 0.395751953125, 0.5234375, 0.9150390625, -0.0419921875, 0.2008056640625, 0.257080078125, -1.298828125, 0.2529296875, 0.307861328125, 1.1103515625, 0.273193359375, 0.97509765625, -1.2265625, -0.08099365234375, 0.59521484375, -0.2349853515625, 0.1409912109375, -0.35400390625, 0.1678466796875, 0.62841796875, -1.4609375, -0.10443115234375, 1.7626953125, 1.5146484375, 1.2080078125, -1.4404296875, 0.62451171875, -1.4755859375, 0.344970703125, -0.8408203125, 0.377685546875, -0.4951171875, 0.33251953125, -0.400146484375, 0.44970703125, -0.444091796875, -1.0146484375, 0.46728515625, 1.3125, -3.12109375, -1.68359375, -1.162109375, -0.1265869140625, 1.9033203125, 0.845703125, -0.5615234375, -1.2275390625, -0.06317138671875, 1.0791015625, 2.34765625, -0.61962890625, 1.212890625, -0.8974609375, -0.08636474609375, 0.90966796875, -1.208984375, 0.1278076171875, 0.0136566162109375, 1.41015625, -0.30029296875, 1.431640625, 0.03851318359375, -0.93603515625, -1.4755859375, -0.1531982421875, -0.423095703125, 0.326171875, -1.224609375, 0.322509765625, -0.4833984375, -0.892578125, -0.483154296875, -0.1126708984375, 2.2578125, 0.3291015625, -0.85595703125, -1.0791015625, -0.1571044921875, 0.55029296875, 1.423828125, 0.1798095703125, 1.0458984375, -0.970703125, -0.031707763671875, -2.578125, -0.8232421875, -2.607421875, 0.64404296875, -0.281005859375, 0.52099609375, -0.11859130859375, 0.748046875, 2.205078125, -0.0810546875, 1.4306640625, 1.7685546875, -1.90234375, -1.4462890625, -0.286376953125, 0.46923828125, -1.8408203125, 0.25732421875, -0.1883544921875, -0.54541015625, -2.15625, 1.6904296875, -1.4599609375, -0.75927734375, 0.475341796875, -1.404296875, -0.46337890625, -0.775390625, -0.458984375, -1.2529296875, 0.25244140625, -0.33544921875, 0.50341796875, 0.2471923828125, 0.53515625, 1.529296875, 1.349609375, 0.75537109375, -0.0843505859375, 1.158203125, -0.59326171875, 1.3828125, 1.24609375, 0.2130126953125, 0.231689453125, -1.384765625, 1.6875, -1.1826171875, 0.221923828125, 0.10369873046875, -0.309814453125, -0.06500244140625, 0.480224609375, -0.5703125, 0.80517578125, 0.2919921875, 0.490966796875, -0.60205078125, -0.1280517578125, 1.244140625, 1.4482421875, 1.0986328125, 1.21875, 0.1968994140625, -1.359375, 0.6904296875, 0.316650390625, -2.171875, 1.0244140625, 1.1025390625, 0.33056640625, 0.9013671875, 0.47412109375, 1.1669921875, 2.322265625, 2.6171875, 0.435791015625, 1.740234375, 2.51171875, 0.2442626953125, -1.3046875, 0.441650390625, 0.36279296875, 0.41845703125, -1.2099609375, -0.60595703125, -0.179931640625, -0.97265625, 0.00353240966796875, 0.455322265625, -1.1083984375, -0.6640625, 0.391845703125, 1.6513671875, 0.50537109375, -0.42919921875, 0.671875, -0.2459716796875, -1.1474609375, -2.30078125, -0.61474609375, -1.2119140625, -3.18359375, -0.221435546875, -0.501953125, -0.81494140625, 0.6884765625, -0.6630859375, -0.39794921875, 2.16796875, 1.8095703125, 1.85546875, 1.140625, 0.186279296875, -0.0595703125, -0.89599609375, -0.2105712890625, -0.35009765625, -1.2529296875, 0.46630859375, 1.2568359375, -0.97900390625, 0.0924072265625, 0.87646484375, 0.418212890625, -0.07244873046875, 0.978515625, -0.4658203125, -0.21875, 0.0207672119140625, -0.10467529296875, 0.5205078125, -1.03515625, -0.98681640625, 0.5615234375, -1.00390625, -0.422607421875, -0.200439453125, -0.84716796875, 0.2249755859375, -0.040802001953125, 1.7041015625, -2.333984375, 1.9873046875, 0.78564453125, -0.57275390625, 2.2734375, -0.654296875, 1.6318359375, -2.595703125, -1.1669921875, -0.59765625, 1.5947265625, 1.2158203125, 0.75390625, 0.390625, -0.47021484375, 0.341064453125, 0.63330078125, 0.160888671875, 0.195556640625, -0.1256103515625, -1.65625, 1.017578125, -0.72509765625, 0.587890625, 0.56640625, -0.5322265625, 0.1051025390625, -0.939453125, 0.113525390625, -0.319580078125, 1.2529296875, 0.7568359375, 0.56640625, 0.203125, 0.57275390625, -0.2247314453125, -1.8271484375, 0.97216796875, -1.0, -0.98779296875, 0.1458740234375, -0.955078125, 2.064453125, -0.96630859375, 0.316650390625, 0.02874755859375, -0.06744384765625, -0.78369140625, -0.9951171875, 0.77587890625, -2.068359375, 0.322265625, -0.17041015625, 0.9052734375, 0.319091796875, 1.4208984375, -1.5771484375, 0.69140625, 0.289306640625, 1.279296875, 1.2958984375, -0.60009765625, 0.7421875, 0.353271484375, -1.435546875, 1.23046875, 1.451171875, -0.7373046875, -0.28466796875, 1.0517578125, -0.67041015625, -0.8955078125, 1.177734375, -1.224609375, 0.1314697265625, 0.5185546875, 2.072265625, -1.0859375, -0.11468505859375, -0.254638671875, 0.48779296875, 0.666015625, 1.3701171875, -1.7822265625, 1.509765625, 1.0185546875, -1.0703125, 0.366943359375, -0.25439453125, -0.859375, -0.382568359375, 0.3916015625, 0.270751953125, -0.9111328125, -0.787109375, 0.09930419921875, -2.021484375, -0.259033203125, 0.5693359375, -0.442138671875, -0.06793212890625, 0.8837890625, 1.0302734375, 0.276123046875, -1.4921875, -1.9248046875, -0.32421875, 1.4609375, -1.0048828125, -0.3779296875, 1.4423828125, 0.96533203125, -1.1884765625, 1.2275390625, 0.51123046875, 0.28271484375, 0.95361328125, 0.11395263671875, 0.88232421875, -2.796875, -0.460693359375, -1.150390625, -0.45849609375, -2.419921875, 0.84326171875, 0.452392578125, -0.174560546875, -0.55712890625, 0.6962890625, -0.53515625, -0.2020263671875, -2.703125, -0.8330078125, 0.5029296875, -1.0, 0.226318359375, 0.88134765625, 1.09765625, 0.6845703125, -0.32470703125, -0.486572265625, 0.306640625, 0.018157958984375, 1.0546875, -0.0120697021484375, -0.82470703125, -1.1552734375, 0.11700439453125, -0.440673828125, 0.95458984375, 0.461669921875, 0.05169677734375, 0.0745849609375, -0.91943359375, -0.76611328125, -0.38037109375, -1.0712890625, -1.177734375, -0.908203125, 1.587890625, 1.19140625, 0.051788330078125, 0.2083740234375, 0.306640625, -0.56689453125, 0.332275390625, -1.533203125, -0.18994140625, 1.2666015625, -0.24072265625, -1.08203125, -1.189453125, 0.79541015625, 0.2122802734375, 0.72021484375, -0.41552734375, -0.383544921875, 0.2081298828125, 0.00460052490234375, 0.326416015625, 0.67626953125, 0.93359375, 0.3359375, 0.740234375, -1.5400390625, -0.6162109375, -3.16796875, -0.2100830078125, -0.93701171875, -1.50390625, 0.08782958984375, 0.330078125, -0.042388916015625, -0.04510498046875, -0.10479736328125, -1.96875, -0.475830078125, -1.53125, 0.16943359375, -1.0439453125, 0.462890625, -1.072265625, 0.57275390625, -0.0782470703125, -0.449951171875, -1.447265625, -0.0924072265625, -1.83203125, 0.7958984375, 2.083984375, 0.341552734375, -0.48291015625, -0.208740234375, -0.8017578125, -0.31982421875, -1.75, 1.0458984375, 0.734375, -3.302734375, -0.308349609375, 0.37646484375, 1.1767578125, -0.287109375, 0.7431640625, -0.1446533203125, 0.398681640625, -1.322265625, 2.0625, 1.0537109375, -1.064453125, -0.355712890625, -2.2578125, -0.335205078125, -1.224609375, -0.1954345703125, 1.9482421875, -0.61181640625, 0.459716796875, -0.62255859375, -0.73486328125, -0.429931640625, 0.69482421875, -1.65234375, -1.0234375, 1.2734375, -1.400390625, -1.6396484375, 0.931640625, -0.91943359375, 0.39111328125, 0.01739501953125, 0.5673828125, 0.04534912109375, -0.9794921875, -0.82373046875, 1.1669921875, 0.308837890625, -1.9052734375, 0.7119140625, 1.1845703125, 0.311767578125, -0.861328125, 1.4912109375, -0.429443359375, -0.59619140625, 0.82470703125, 1.375, -0.8740234375, -1.5576171875, 1.1259765625, -0.36572265625, -1.5654296875, -0.59228515625, 0.52685546875, 1.041015625, 0.237548828125, 0.056396484375, 1.7607421875, -0.77099609375, -1.275390625, 0.86572265625, -1.4013671875, -0.91552734375, -0.47509765625, -1.474609375, -0.5810546875, -0.65576171875, 0.476806640625, -1.7783203125, -2.298828125, 0.054534912109375, 0.494873046875, -1.24609375, 0.222412109375, -1.6337890625, -0.0198516845703125, -0.221923828125, -0.1427001953125, -0.330322265625, 1.2490234375, -0.8935546875, -0.64306640625, 0.0750732421875, 0.962890625, -0.39111328125, 0.27734375, -0.3662109375, -0.0955810546875, -1.5439453125, -0.1689453125, 1.2900390625, -0.216796875, 0.02337646484375, 1.576171875, 0.312255859375, -0.00861358642578125, 0.90478515625, -0.4306640625, -0.77685546875, -0.284912109375, 0.892578125, -0.31640625, -1.1953125, 0.452880859375, 0.40625, 0.7236328125, 1.1640625, 0.08526611328125, -0.58984375, -1.06640625, 0.88427734375, 0.11981201171875, 0.400146484375, 0.625, -0.9423828125, -1.7490234375, 0.75390625, -0.79248046875, 0.72216796875, 0.615234375, -1.033203125, 1.0732421875, -0.36767578125, -1.939453125, 0.448486328125, 0.058624267578125, 0.6171875, 1.326171875, -0.505859375, -1.3330078125, 0.5830078125, -0.410888671875, 1.3046875, -1.7265625, -1.3173828125, 0.8759765625, -0.2236328125, -0.9169921875, 1.224609375, -0.71240234375, 0.8125, 0.10430908203125, -1.1357421875, -0.60498046875, -0.56005859375, 0.92041015625, -1.0283203125, -1.2001953125, 0.861328125, -1.16796875, 2.1328125, -0.10662841796875, 2.0390625, 0.282470703125, -0.724609375, 0.5791015625, -0.0001169443130493164, 0.460205078125, -0.634765625, 0.861328125, -1.0185546875, 0.52685546875, 0.11578369140625, 0.6630859375, -1.3115234375, 1.1640625, 0.8056640625, 0.043365478515625, 0.1756591796875, 2.03515625, 1.71875, -0.4033203125, -0.296630859375, 0.094970703125, 2.19921875, -0.818359375, -0.95263671875, -1.6953125, 0.537109375, -1.154296875, -0.96435546875, 0.24560546875, 0.232177734375, -0.7607421875, -0.8251953125, -0.0684814453125, -2.4453125, 1.0185546875, 0.095947265625, -0.17578125, -0.49072265625, -2.095703125, -0.77197265625, -1.388671875, 0.489501953125, 1.4716796875, -0.10687255859375, -0.11151123046875, -0.0426025390625, 1.2861328125, -0.583984375, 0.31884765625, 1.263671875, -0.397705078125, -1.0439453125, 0.267333984375, -0.57568359375, 0.95263671875, 0.623046875, 1.283203125, 1.7158203125, -1.642578125, -0.469970703125, -0.09857177734375, -0.96826171875, -0.8486328125, 1.537109375, 1.037109375, -0.59326171875, -0.1773681640625, 0.06817626953125, -1.017578125, 0.71630859375, -0.1689453125, 1.064453125, -1.0537109375, 0.442138671875, -0.244140625, 0.55126953125, 0.07562255859375, -0.2373046875, -1.2841796875, 1.0498046875, -0.09136962890625, 0.301513671875, -0.2064208984375, 0.8515625, -0.8525390625, 0.07427978515625, 0.6611328125, 0.08807373046875, -0.82763671875, 0.21484375, -0.86572265625, -0.333984375, -0.50048828125, -0.1317138671875, -1.19140625, -1.21875, -0.8212890625, -0.143310546875, 0.29638671875, 0.96240234375, 1.06640625, -0.0439453125, -0.60791015625, 1.3935546875, -0.68359375, -0.64501953125, -0.321044921875, 0.828125, -1.0234375, 0.291748046875, 0.73876953125, -1.1083984375, 0.95458984375, 0.75927734375, -0.4814453125, -0.77197265625, 0.11749267578125, 0.058685302734375, -0.8994140625, 0.0877685546875, -1.173828125, -0.8232421875, 1.39453125, 0.449462890625, 0.544921875, 0.57080078125, 1.33984375, 2.427734375, -1.2392578125, -0.80810546875, -0.25244140625, -1.09765625, -1.4033203125, 0.640625, -0.27001953125, -0.49609375, -0.4853515625, -1.2705078125, 0.3505859375, -0.564453125, -1.3037109375, -1.3642578125, 0.57373046875, 0.6435546875, 0.2186279296875, 0.6396484375, -1.373046875, 0.4228515625, 0.1905517578125, 1.1669921875, -0.2386474609375, 0.0455322265625, 0.6875, -0.385498046875, -0.96142578125, -0.796875, 1.7451171875, 0.81396484375, -1.515625, 0.94140625, 0.09295654296875, 0.755859375, 0.1329345703125, 0.3876953125, -0.66357421875, 1.9443359375, -0.66162109375, -0.9169921875, 0.14306640625, 1.609375, 0.630859375, 0.48388671875, 0.06927490234375, -1.787109375, -0.23486328125, 0.623046875, -1.67578125, 0.1971435546875, -0.175048828125, 0.12261962890625, 0.5234375, 0.7978515625, 0.92041015625, 0.58154296875, 0.78955078125, -0.317626953125, 0.25439453125, -0.93505859375, -0.08673095703125, -0.5791015625, -1.890625, 0.68701171875, -1.1416015625, 0.28564453125, -1.5361328125, -0.08160400390625, 0.377685546875, 2.025390625, 1.6220703125, 0.615234375, -0.1572265625, -1.234375, -0.96630859375, 0.71923828125, 0.96875, -0.31201171875, -1.0126953125, 0.6298828125, -0.95751953125, 1.5390625, 0.98974609375, -0.85009765625, 1.4599609375, -0.326171875, -0.8388671875, -0.9443359375, -0.361328125, -0.97607421875, 0.77294921875, -0.8466796875, 0.23779296875, -0.236328125, 1.2958984375, 0.98291015625, 1.822265625, 0.466796875, 0.341064453125, 1.3037109375, 0.2374267578125, -0.127685546875, 0.1405029296875, -0.475341796875, 1.5703125, -1.216796875, -0.029144287109375, -0.281494140625, -0.1173095703125, -1.3310546875, 0.73388671875, -1.5947265625, -0.54296875, 0.5419921875, 0.49365234375, 1.0048828125, -0.56884765625, -1.2529296875, -1.6240234375, 0.69482421875, 1.0068359375, -0.404541015625, 0.1429443359375, -0.429931640625, 0.473388671875, -0.2132568359375, 0.64990234375, -0.01593017578125, 0.1627197265625, -1.30078125, -0.389892578125, 1.88671875, -0.40576171875, 0.87060546875, -0.892578125, 0.75, -0.359375, 1.00390625, -0.58935546875, -1.0400390625, 0.0682373046875, 1.33984375, 1.625, 0.513671875, 0.94384765625, 0.31103515625, -0.373046875, -0.1905517578125, 0.51220703125, 0.07330322265625, -0.458984375, 0.7900390625, 0.25341796875, -0.295166015625, -1.1416015625, 0.7509765625, 1.3056640625, 1.8212890625, 0.0173187255859375, -0.396484375, -1.2431640625, 0.7177734375, -2.048828125, -2.5078125, 0.1732177734375, -0.55615234375, 1.3505859375, -0.50048828125, 0.12432861328125, -0.76611328125, -0.6259765625, -0.4697265625, -1.19140625, 0.65673828125, -0.414306640625, -1.3583984375, 1.4326171875, 0.47900390625, -0.1634521484375, -1.6328125, 0.16259765625, 0.172119140625, 0.05975341796875, 0.15966796875, -0.818359375, -0.432861328125, 2.482421875, -0.45947265625, -0.0965576171875, -1.76171875, 1.58984375, 0.69189453125, 0.04150390625, -0.5830078125, -2.267578125, 0.7041015625, -0.255859375, -2.162109375, 0.04754638671875, 0.81396484375, -0.1572265625, 0.286376953125, 0.28759765625, 0.9609375, 0.263916015625, -0.73095703125, 0.30419921875, 1.9775390625, -0.98974609375, -0.60302734375, 1.2802734375, -0.62744140625, 0.81396484375, -0.78125, 1.298828125, -0.701171875, 1.9775390625, -0.5009765625, 1.3271484375, -1.39453125, -0.591796875, -0.95947265625, -0.63720703125, -0.048583984375, -0.47021484375, -1.802734375, 1.0146484375, 1.2041015625, 0.6845703125, -0.339599609375, 0.4287109375, 1.6767578125, 1.2490234375, -0.5595703125, 2.064453125, -0.392578125, 1.6787109375, -0.1337890625, -0.22705078125, -0.703125, 2.646484375, 1.68359375, -0.80908203125, 0.5361328125, -0.53662109375, 0.68798828125, 0.7490234375, 0.254638671875, 1.5478515625, -1.5322265625, -1.4697265625, 1.3515625, -0.25048828125, 1.400390625, -0.2225341796875, 0.97412109375, 1.16015625, -0.40673828125, 0.85498046875, 0.91162109375, -1.056640625, 1.220703125, 1.19921875, 1.71875, -1.005859375, -1.2880859375, -0.101806640625, -0.70068359375, 0.61962890625, -0.77392578125, 1.423828125, -0.7265625, 1.0458984375, -0.67724609375, 0.86376953125, 0.9892578125, 0.671875, 0.412841796875, -0.7705078125, 1.0126953125, -0.363037109375, -0.383056640625, -0.57470703125, -0.59228515625, 0.131103515625, 0.1439208984375, -0.53564453125, 1.072265625, 1.34765625, -0.2166748046875, -0.379638671875, -0.2744140625, 0.3779296875, 0.61669921875, 0.0968017578125, -0.35107421875, -0.591796875, 0.95751953125, -0.2322998046875, -1.091796875, -1.6328125, 0.57177734375, -1.341796875, 1.1767578125, 0.11181640625, 1.7529296875, 1.76953125, 1.669921875, 0.86962890625, -0.53271484375, -1.7568359375, 0.29052734375, 0.27294921875, -0.5244140625, 1.076171875, -0.192138671875, -1.212890625, -0.8603515625, 0.287109375, -1.0791015625, -0.1634521484375, -2.1796875, -0.60498046875, -0.038421630859375, -0.9873046875, -1.818359375, -0.80029296875, 0.2369384765625, -1.900390625, -1.3916015625, -2.00390625, 1.4658203125, -0.9462890625, -0.8466796875, -2.337890625, 0.235595703125, 0.72314453125, -0.1292724609375, 1.740234375, -1.16796875, -0.85546875, -0.075439453125, -1.076171875, 0.29833984375, -0.09014892578125, -0.387939453125, -0.72607421875, -0.341064453125, -0.7431640625, 0.892578125, -0.283935546875, 1.283203125, 1.5556640625, 0.295166015625, -1.0185546875, -1.453125, 0.6240234375, -0.01105499267578125, -0.420654296875, 0.892578125, 1.822265625, 0.37353515625, -1.505859375, 0.0987548828125, 0.446044921875, 0.68359375, -0.97607421875, -0.08685302734375, 1.7021484375, -1.8740234375, -0.953125, 0.013214111328125, -1.853515625, 0.1768798828125, -0.288818359375, -0.11358642578125, -0.84228515625, 0.93115234375, -1.8642578125, -0.501953125, -0.427734375, 0.69580078125, -0.7724609375, 0.86669921875, 0.493896484375, -1.57421875, -0.0960693359375, 1.9033203125, -0.79052734375, -0.41455078125, 0.517578125, 0.63623046875, -0.57275390625, -1.001953125, -1.421875, 0.51513671875, -1.384765625, 0.78173828125, 0.395751953125, -0.5771484375, 0.67529296875, -0.134521484375, -0.273193359375, -0.8046875, -0.68310546875, -0.400146484375, 1.6875, -0.5439453125, 1.205078125, 0.580078125, 1.0849609375, 2.564453125, -1.7255859375, -0.7265625, -0.58935546875, 0.18603515625, 0.82421875, -0.720703125, 0.705078125, -0.70458984375, -0.07049560546875, 0.126953125, 1.1865234375, 0.344970703125, 0.8701171875, 0.626953125, 1.515625, 1.435546875, -0.27783203125, 1.185546875, -0.4619140625, -1.6474609375, 0.27685546875, 0.732421875, -0.06256103515625, 2.37109375, 0.48681640625, 0.384765625, 1.751953125, -1.1591796875, 0.2430419921875, -0.83984375, 0.12115478515625, 0.541015625, 0.361083984375, 3.177734375, -0.10845947265625, -1.1416015625, -1.466796875, 0.1292724609375, 1.2275390625, -0.432861328125, 0.90380859375, 0.66357421875, 0.86279296875, 0.3515625, 0.67724609375, -1.166015625, 0.5166015625, -2.353515625, -1.18359375, 1.6591796875, -0.71630859375, -2.462890625, -1.00390625, -0.0389404296875, -0.9658203125, -1.8896484375, 0.5830078125, 1.42578125, -0.669921875, -0.1995849609375, -0.0030918121337890625, -1.576171875, -0.09381103515625, 1.9775390625, 1.6494140625, -0.240234375, -1.1484375, 0.055816650390625, -2.734375, -0.9990234375, -1.57421875, 0.04290771484375, 0.77294921875, 1.4931640625, -0.59521484375, -1.3193359375, -0.2099609375, 2.47265625, 0.6591796875, -1.2802734375, -0.8203125, 0.70068359375, -1.3173828125, -0.443115234375, 1.005859375, 0.8515625, 0.923828125, -0.744140625, -1.01953125, -0.59521484375, -0.02783203125, 1.630859375, -1.29296875, 1.3486328125, 1.37109375, 0.0716552734375, 0.85107421875, -1.609375, 0.08544921875, 1.298828125, -0.11175537109375, -0.74853515625, 0.8984375, -0.43603515625, 1.3359375, 1.8310546875, -0.323486328125, -0.412353515625, -1.611328125, 0.50927734375, 0.69775390625, -0.97607421875, 0.3486328125, 1.2626953125, 1.2421875, 3.111328125, -1.146484375, 0.734375, 1.134765625, -1.416015625, 0.0850830078125, -1.400390625, -1.6904296875, 0.3876953125, 1.142578125, 1.1064453125, -1.5498046875, -0.94189453125, -0.99755859375, -0.43017578125, 0.30810546875, -2.46484375, -2.2265625, 0.6689453125, 1.5244140625, 0.15478515625, -0.1402587890625, -1.552734375, -0.84375, -1.357421875, -0.3310546875, 0.411865234375, -0.10150146484375, -1.1279296875, -1.2138671875, 1.552734375, -0.0012273788452148438, -0.488525390625, 0.25927734375, 1.2138671875, -1.0595703125, 0.97265625, 1.119140625, 0.366943359375, -1.7568359375, 1.1123046875, 1.1630859375, 1.2734375, 2.609375, 0.225830078125, 0.477783203125, 0.482421875, -1.533203125, -0.90625, -0.8134765625, -0.01139068603515625, -0.430419921875, -0.1136474609375, -0.03558349609375, -0.8173828125, -0.1031494140625, -1.876953125, 1.478515625, 0.69091796875, 0.82958984375, 0.06927490234375, 0.24267578125, 0.61474609375, -0.828125, 0.0836181640625, 0.720703125, 0.77587890625, -0.065673828125, -1.876953125, 1.6875, 0.66015625, -1.05859375, 0.79638671875, 0.037017822265625, 0.00409698486328125, 0.0188751220703125, -0.681640625, -1.9150390625, -0.984375, 0.414794921875, -0.5322265625, 0.8701171875, 0.9697265625, 0.030181884765625, 0.55322265625, 0.0753173828125, -0.796875, -0.6142578125, -0.4794921875, 1.12109375, -0.81494140625, -0.68896484375, -0.174560546875, -1.42578125, 0.109619140625, 0.37841796875, -0.83251953125, -0.82177734375, -1.263671875, -0.5517578125, -1.57421875, 0.35107421875, -0.26416015625, -0.99658203125, 1.5791015625, -0.00531768798828125, -0.75537109375, -3.255859375, 0.39697265625, -2.33203125, -0.720703125, 0.51416015625, 0.455810546875, -1.51953125, -0.85693359375, 0.62255859375, -0.16943359375, 1.7861328125, -0.05975341796875, 1.685546875, -1.060546875, 0.513671875, 0.9951171875, 0.11767578125, 0.6845703125, -0.55517578125, 0.2587890625, 0.2349853515625, -0.8828125, 0.93408203125, 0.91455078125, 0.331787109375, -0.650390625, 0.50439453125, 0.024688720703125, 0.53173828125, -0.82763671875, -0.3212890625, 0.2265625, 2.2109375, 0.10992431640625, -0.671875, -0.22021484375, 0.927734375, 0.2125244140625, 1.0625, -0.328857421875, 0.6240234375, -0.0745849609375, 1.0341796875, -1.4814453125, -0.426513671875, 1.12109375, -1.134765625, -0.68798828125, -1.7109375, 0.1973876953125, -0.06646728515625, -0.85693359375, 0.467041015625, 0.7109375, -0.521484375, 1.45703125, 0.79345703125, 0.09625244140625, 0.340087890625, 2.337890625, -0.90380859375, 0.9912109375, 0.007556915283203125, -2.244140625, 0.1492919921875, 1.3212890625, -0.33837890625, 1.1455078125, -0.94873046875, -0.9375, -0.1522216796875, -0.52587890625, -0.04034423828125, 2.134765625, 0.234130859375, 1.0556640625, -0.83642578125, -2.408203125, 0.744140625, 1.1083984375, -0.62158203125, -0.462646484375, -0.26220703125, -1.873046875, 0.41796875, 1.0830078125, 1.0029296875, 0.61474609375, -0.2086181640625, -0.1597900390625, -0.0254669189453125, 0.75634765625, -0.9423828125, -0.66064453125, 0.333251953125, 0.021514892578125, -0.57275390625, -0.12384033203125, 0.013763427734375, 0.68115234375, 2.8203125, 0.76171875, -0.006053924560546875, -0.423095703125, -0.7431640625, -0.326171875, 0.70361328125, -1.7421875, -0.68017578125, 1.234375, 1.521484375, -1.5732421875, -0.9296875, -1.0556640625, -1.0380859375, -3.501953125, 0.84619140625, 1.416015625, 0.4599609375, -1.2666015625, 1.576171875, -0.033416748046875, 2.61328125, 0.00647735595703125, -0.8232421875, -0.548828125, 0.54931640625, -0.405517578125, 0.95556640625, -0.129150390625, -0.74560546875, -1.08203125, -0.712890625, 0.29296875, 0.66162109375, 1.4150390625, 1.9931640625, -1.9736328125, 1.3427734375, -0.08038330078125, -1.6142578125, -0.003387451171875, -1.7275390625, 0.269775390625, -0.9541015625, -0.9560546875, 0.09674072265625, -0.479736328125, -1.64453125, 0.8369140625, 0.07275390625, -3.537109375, -0.053497314453125, -0.1412353515625, -0.16748046875, -0.10064697265625, 0.22119140625, -0.0012445449829101562, -0.60693359375, -0.611328125, 1.322265625, -1.5322265625, -1.380859375, -0.58984375, 1.2626953125, -0.281005859375, 1.462890625, -0.304931640625, -0.403076171875, -1.076171875, -0.75390625, -0.99072265625, -0.1444091796875, 0.14599609375, -0.60205078125, 1.3662109375, -0.2626953125, 0.62353515625, 1.751953125, -1.08203125, 0.93408203125, -0.1864013671875, 1.4306640625, 1.794921875, 0.1781005859375, 0.06060791015625, -0.1702880859375, 0.64892578125, -0.78173828125, -0.212158203125, -1.91015625, -0.55126953125, -0.62158203125, 1.0537109375, 1.16796875, -0.407470703125, 0.8193359375, 0.767578125, 0.95068359375, -1.1708984375, -0.71728515625, -0.82958984375, -0.390625, -1.5849609375, -0.9208984375, 1.2138671875, 0.69287109375, -0.406982421875, 1.337890625, 0.054840087890625, 0.1387939453125, -0.376708984375, -1.2294921875, -0.79931640625, -0.5126953125, -0.0875244140625, -0.97900390625, -0.66650390625, 0.6201171875, 1.98828125, -0.5322265625, 0.1783447265625, -2.373046875, 2.212890625, 1.4404296875, 0.87890625, -1.046875, -0.72705078125, -1.5927734375, 0.27294921875, 0.0714111328125, 0.6259765625, -1.5244140625, 0.31640625, 2.345703125, -1.255859375, 0.173095703125, -0.379638671875, -1.0791015625, -0.5703125, 0.5927734375, -0.1123046875, 1.2236328125, 1.109375, 0.72314453125, -1.2744140625, -0.019287109375, 0.80322265625, 0.228759765625, 0.3740234375, -0.94580078125, 1.6728515625, -1.619140625, 0.10797119140625, -0.568359375, -0.390869140625, 1.6904296875, -1.119140625, 1.7587890625, -0.444091796875, 0.63037109375, 1.2265625, 0.57958984375, -1.1279296875, -0.20458984375, -0.6171875, -0.336669921875, -0.1871337890625, -0.79931640625, -0.236572265625, 1.28125, 1.0244140625, -0.308837890625, 0.106201171875, 0.77197265625, -1.4462890625, -1.2158203125, -0.276611328125, -0.2296142578125, -0.353759765625, -0.251953125, -1.5703125, -0.6865234375, -1.6962890625, 1.625, 0.00262451171875, -0.10888671875, -1.0556640625, 1.3330078125, 1.2392578125, 0.73486328125, 0.38623046875, 0.09783935546875, 1.3828125, 2.302734375, -1.5546875, 0.0767822265625, 0.6357421875, 0.2376708984375, 0.97998046875, 0.416015625, -0.8310546875, -0.0117340087890625, 0.32177734375, -0.59521484375, -1.041015625, -1.234375, -0.65673828125, -1.908203125, 0.42578125, -1.4365234375, 0.54150390625, -0.958984375, 1.5126953125, 1.1494140625, 0.0352783203125, -0.7919921875, 1.6220703125, -0.320068359375, -0.85205078125, 0.8642578125, -0.430419921875, -0.00902557373046875, -0.1412353515625, -0.253662109375, 1.04296875, 0.50634765625, 0.09210205078125, -0.1683349609375, -0.51220703125, -0.1590576171875, 0.6943359375, -0.376708984375, -1.42578125, -0.43701171875, -0.359375, -1.509765625, 0.09783935546875, 0.030029296875, -1.734375, -1.203125, 0.416259765625, -0.26904296875, 0.301025390625, -0.2076416015625, 0.89306640625, 0.428955078125, 0.32763671875, 0.52783203125, 1.2861328125, -0.051971435546875, 0.1400146484375, 0.541015625, -1.2275390625, 1.5498046875, 0.78759765625, 1.1640625, 0.61962890625, 0.61083984375, 0.21630859375, 0.369384765625, -0.83837890625, -0.358642578125, 0.0340576171875, -0.908203125, -0.79248046875, 0.83251953125, -0.6240234375, 0.03216552734375, -0.275390625, -1.0654296875, 0.300048828125, 0.80615234375, 1.337890625, -0.75, -0.7041015625, -1.251953125, -1.3291015625, 1.4521484375, 1.0166015625, -0.347900390625, 3.22265625, 0.6962890625, 0.8193359375, 0.1029052734375, 0.9189453125, 0.495361328125, 0.0526123046875, 0.73876953125, -0.41552734375, 0.65234375, -0.98583984375, -1.33203125, 0.325927734375, -0.9677734375, -0.307861328125, 0.1923828125, 0.6220703125, -0.5361328125, 0.5830078125, -2.015625, 1.5283203125, 0.7822265625, -0.650390625, -0.005817413330078125, 0.4541015625, 1.2890625, 0.83251953125, 0.231201171875, -0.25, 0.53466796875, 0.80029296875, -0.392578125, -0.0870361328125, -1.0400390625, 0.3701171875, -0.252685546875, -0.02459716796875, 0.09051513671875, -0.41796875, -0.83251953125, 0.55810546875, -0.6962890625, -0.2005615234375, 1.3212890625, -1.4111328125, -0.81982421875, -0.60302734375, -0.22314453125, 1.86328125, 0.367431640625, 0.0175323486328125, -0.3203125, 0.91259765625, 0.8291015625, -1.177734375, 1.8369140625, 1.458984375, 0.57421875, 0.052764892578125, 1.3232421875, 0.56494140625, -0.48828125, -0.9169921875, -1.0498046875, 0.22509765625, -0.78271484375, -0.4189453125, 0.65234375, -1.5458984375, -0.8212890625, 0.267578125, -0.8798828125, 1.3779296875, -0.03424072265625, -0.58154296875, 2.111328125, -1.8525390625, 1.673828125, -0.0810546875, 0.55322265625, -1.1474609375, 0.8076171875, 0.20166015625, 0.004241943359375, -0.8349609375, 1.5166015625, 0.759765625, -0.0865478515625, -0.87548828125, 1.205078125, 0.015838623046875, -1.1171875, 1.0869140625, -0.61279296875, -0.3681640625, 1.5869140625, -0.281005859375, -0.34521484375, 1.064453125, -0.86669921875, -0.50390625, -0.10272216796875, -0.66650390625, 1.34765625, -0.30615234375, -0.4267578125, 0.91259765625, -1.33984375, -0.62841796875, 1.1025390625, -1.6572265625, 1.37109375, 0.412841796875, -0.10723876953125, -0.0005784034729003906, 0.59033203125, 0.63330078125, -1.82421875, 0.3076171875, -0.254638671875, -0.058197021484375, 0.51416015625, 1.095703125, -0.611328125, -1.095703125, 0.6748046875, 0.44482421875, 0.8544921875, -1.259765625, 1.2080078125, 1.5439453125, -0.50830078125, 0.463134765625, -0.8125, 1.7158203125, -1.794921875, -0.060394287109375, 0.8857421875, 0.693359375, -0.1256103515625, -0.225830078125, 0.094970703125, -0.1131591796875, -0.64697265625, -0.859375, -0.00783538818359375, -0.0860595703125, -1.390625, -0.521484375, -3.23046875, -0.244384765625, 0.67333984375, -0.294921875, 0.221923828125, -0.84619140625, -0.2103271484375, 0.6201171875, -1.0693359375, 2.37109375, 0.94677734375, -0.32568359375, -1.2939453125, -1.078125, 1.1767578125, -0.65576171875, 1.205078125, -0.464599609375, 0.70068359375, -1.0625, -0.595703125, 0.92724609375, 0.48046875, -1.1806640625, 0.3681640625, -0.58349609375, -1.744140625, -0.341796875, -0.814453125, -0.43408203125, -0.361328125, -0.034912109375, -0.0697021484375, 0.78369140625, -0.0107421875, 1.3330078125, 1.3642578125, 0.279296875, 0.59619140625, -0.1256103515625, 1.5166015625, 0.63671875, 1.8984375, -0.57177734375, 1.046875, -0.349365234375, -0.498779296875, 0.04364013671875, -0.3251953125, -0.315673828125, 0.0232696533203125, 0.5380859375, 0.75048828125, -2.30078125, -1.3232421875, 1.0234375, -1.7939453125, 1.36328125, 0.7314453125, -0.1407470703125, -0.228759765625, -1.740234375, 1.0693359375, -0.209716796875, 0.86572265625, 0.9560546875, 0.24755859375, -0.73193359375, 0.28466796875, 1.201171875, 0.09332275390625, 0.962890625, -0.72998046875, -0.97802734375, -0.369873046875, 0.7470703125, 0.06494140625, 0.91455078125, -0.2239990234375, 0.07940673828125, 0.37939453125, 0.083984375, -0.53515625, -0.4228515625, 0.77294921875, 1.357421875, -1.76171875, -0.06793212890625, -0.5224609375, 1.3603515625, 0.63818359375, -0.340576171875, -0.93994140625, 0.00011831521987915039, -0.2440185546875, 0.49072265625, 0.77197265625, 0.178955078125, 0.9951171875, 1.048828125, 0.27197265625, 1.259765625, -0.394287109375, -1.3759765625, -0.9599609375, -0.08758544921875, 0.11956787109375, 0.66162109375, -0.5712890625, -0.43310546875, -0.3056640625, 0.83349609375, 0.42822265625, 0.119384765625, -0.430419921875, -0.2169189453125, 1.76953125, -0.92431640625, 1.1689453125, -0.70849609375, 0.75927734375, 0.67724609375, 0.68896484375, 0.031219482421875, 0.418212890625, -1.80859375, 0.307861328125, 0.74267578125, 0.7392578125, -1.677734375, 1.0595703125, -0.84619140625, 0.7431640625, 0.06903076171875, 0.287109375, 0.5078125, -0.6025390625, 0.76513671875, 0.479248046875, -0.998046875, -1.52734375, -0.94970703125, 0.552734375, 2.017578125, -0.93310546875, 0.85693359375, 0.7822265625, -0.1556396484375, -0.99462890625, 1.25, 0.80029296875, 0.7509765625, 0.70703125, 1.404296875, 0.76416015625, -0.08111572265625, 0.69580078125, 0.34228515625, 0.1878662109375, 0.57275390625, -0.10595703125, 0.509765625, 0.291259765625, 0.2369384765625, -0.236083984375, 1.033203125, 0.7919921875, 0.295654296875, -1.6416015625, -0.123046875, -0.77099609375, 0.30029296875, -0.029815673828125, 0.8203125, 0.281494140625, 0.06353759765625, -0.191162109375, 0.8349609375, -0.57373046875, -1.0546875, 2.076171875, 0.363525390625, -0.04693603515625, -0.93896484375, 1.8837890625, -0.48291015625, 1.5830078125, -2.556640625, 1.5341796875, 1.181640625, 0.32958984375, 0.45703125, -1.8447265625, 0.494140625, -0.0699462890625, 2.642578125, -0.4228515625, 0.85302734375, 0.0194549560546875, 0.249267578125, 0.75244140625, -1.4150390625, 0.1746826171875, 1.8427734375, 0.6865234375, 0.58740234375, 0.336181640625, 1.2314453125, 0.02093505859375, -0.802734375, 0.318359375, 0.63818359375, -0.509765625, 0.37646484375, -0.470703125, -0.341064453125, 1.916015625, 0.173095703125, -0.373046875, -0.459228515625, 0.453369140625, -1.4052734375, 0.9736328125, -0.3955078125, 1.24609375, -1.1279296875, 1.494140625, 0.01800537109375, -1.4375, 1.4150390625, 0.65380859375, -0.08062744140625, 2.3984375, -1.6904296875, 1.6044921875, 0.44775390625, 0.7197265625, -0.205322265625, -1.5029296875, -0.54833984375, -0.5927734375, 0.8232421875, 0.334228515625, 0.2060546875, -1.06640625, -1.3232421875, -0.462890625, 0.266845703125, -0.1363525390625, 0.455078125, 0.292236328125, -2.275390625, 0.2464599609375, -1.3984375, -0.2890625, 0.91796875, -2.44140625, -0.57177734375, 1.041015625, -0.4091796875, 0.1912841796875, 0.1715087890625, -0.72314453125, -1.0966796875, 1.27734375, 0.060760498046875, 1.90625, 2.6875, 0.4677734375, 0.3134765625, 0.315185546875, -0.0384521484375, 0.3974609375, -0.20654296875, 0.0309295654296875, -0.11395263671875, -0.712890625, 0.8046875, -0.60205078125, 1.3544921875, 1.5751953125, -0.4443359375, 0.371337890625, -0.338623046875, 1.5009765625, 0.55615234375, -1.3232421875, 1.171875, 0.09796142578125, -0.947265625, -0.90380859375, -1.0537109375, 0.0347900390625, -0.56689453125, -1.2109375, 0.55126953125, -1.2666015625, -0.5654296875, 0.299072265625, 2.076171875, -0.498779296875, 0.478515625, -0.2247314453125, -0.69677734375, -0.97509765625, 1.0986328125, -0.59375, 0.55712890625, 0.410400390625, 0.61572265625, -1.873046875, -0.958984375, -0.3837890625, 0.53466796875, 0.70751953125, -0.1280517578125, -0.43212890625, 0.58349609375, -0.89599609375, -0.84521484375, 1.1435546875, -0.422119140625, 0.7255859375, -0.62939453125, -1.0830078125, 0.52490234375, -1.318359375, 0.86181640625, -1.7509765625, -0.5556640625, 2.04296875, -0.474365234375, -0.5859375, -0.378662109375, 0.44580078125, -0.1151123046875, -1.4453125, -0.1630859375, -0.323974609375, -0.11248779296875, 1.501953125, -0.2476806640625, -0.33642578125, 0.1568603515625, 0.80224609375, -0.6044921875, 1.9609375, -0.06634521484375, 0.356201171875, -0.1993408203125, 0.7939453125, 1.1455078125, 0.92822265625, -1.0791015625, 1.365234375, 1.005859375, 1.2880859375, -1.96484375, -1.4541015625, 0.35107421875, -1.3896484375, -0.39404296875, -0.95849609375, 0.78662109375, 0.494140625, 0.09112548828125, 1.2666015625, 0.73779296875, -0.5595703125, -0.218505859375, 1.6123046875, -0.291748046875, -0.0279083251953125, -0.56982421875, -0.04083251953125, -0.78466796875, -1.3037109375, 0.036407470703125, -1.767578125, 1.4755859375, -0.1712646484375, -0.77587890625, -2.037109375, 0.035308837890625, 0.131103515625, -0.1673583984375, 0.1885986328125, 0.3095703125, 0.432373046875, -0.387939453125, -0.136962890625, 0.0088348388671875, 0.9501953125, 1.267578125, 0.265625, -0.298828125, 0.67041015625, -1.802734375, -2.216796875, -1.2685546875, 1.548828125, 0.36376953125, -0.343017578125, 0.460205078125, -0.82421875, -1.22265625, 0.158447265625, 0.337890625, 0.67529296875, 1.744140625, -0.81005859375, -0.138427734375, 0.396728515625, 0.58740234375, -0.182861328125, 1.689453125, 0.129150390625, 1.583984375, 0.57958984375, -1.357421875, 0.81103515625, -0.4453125, -0.2137451171875, -0.005443572998046875, -0.15283203125, 0.394775390625, 0.7216796875, -0.54345703125, -1.0048828125, 0.7890625, -0.13232421875, -1.392578125, -0.68017578125, -1.0244140625, -0.237060546875, -1.169921875, 1.5087890625, -0.74169921875, -1.3505859375, -0.6455078125, 0.537109375, 0.5849609375, 0.95458984375, 1.142578125, 0.35888671875, -0.059234619140625, -1.38671875, 0.8154296875, 1.3681640625, -1.390625, 2.75390625, -0.00627899169921875, -1.0537109375, -0.533203125, 0.04736328125, -0.4306640625, 0.5224609375, -2.095703125, -1.55078125, 1.708984375, -0.369140625, -2.244140625, 0.77197265625, 0.904296875, 1.8408203125, 0.94287109375, 1.58984375, 1.3232421875, 3.068359375, 0.4013671875, -1.5234375, -1.4287109375, 0.081298828125, 0.59619140625, 1.234375, -1.169921875, 1.3935546875, -1.2109375, -0.0711669921875, -0.7802734375, -1.4921875, 1.0419921875, 0.98779296875, 0.06890869140625, 1.044921875, -0.943359375, -0.517578125, 0.35009765625, 3.1171875, -0.12249755859375, 0.172607421875, -1.0966796875, 0.92236328125, -0.0270843505859375, 0.2236328125, -1.337890625, 0.83056640625, -0.381591796875, -2.72265625, -1.4521484375, 0.6630859375, -0.65283203125, -1.6787109375, -0.040740966796875, -0.13037109375, 0.767578125, -2.130859375, -0.1732177734375, -0.0863037109375, 2.357421875, -0.08935546875, -1.16796875, -1.5869140625, 0.13134765625, 0.32470703125, -0.8525390625, 0.2113037109375, 0.359130859375, 0.89501953125, -0.11236572265625, -0.1807861328125, 0.90185546875, -1.78125, 0.72705078125, -0.061798095703125, -0.1517333984375, 1.0361328125, 0.2301025390625, -0.423095703125, 0.431640625, -0.303466796875, 1.8720703125, -0.935546875, -0.04644775390625, -2.1015625, -1.09375, -0.58056640625, -0.07672119140625, -0.10284423828125, -1.0791015625, -1.6513671875, -0.494140625, 0.88330078125, 0.6474609375, -1.3720703125, -0.43603515625, 0.63818359375, -0.69482421875, -0.1475830078125, -0.85009765625, -0.27783203125, -0.9345703125, -0.55615234375, -1.5537109375, 0.2105712890625, -3.142578125, 0.873046875, 0.37841796875, -1.2421875, -1.486328125, -1.107421875, 0.13232421875, 0.7138671875, -0.29736328125, 1.82421875, 0.61376953125, -0.357177734375, 0.54736328125, -2.337890625, -0.52197265625, 2.2734375, -1.24609375, 2.044921875, 1.505859375, 0.31884765625, -0.57177734375, 0.65185546875, -0.03857421875, 0.193359375, -0.56396484375, 1.3037109375, -0.168701171875, -0.53564453125, -1.271484375, -0.115234375, -0.1651611328125, -0.06982421875, -1.775390625, 0.79052734375, 1.1279296875, 1.380859375, -1.0712890625, 0.11102294921875, 0.149658203125, -0.7529296875, 1.0478515625, 0.29052734375, 0.143310546875, -0.470947265625, -0.66162109375, 1.056640625, 0.900390625, 0.5029296875, 0.88037109375, 0.312744140625, 0.76123046875, 0.158935546875, 0.359375, 0.83056640625, 1.197265625, 1.7421875, -0.427734375, 0.456298828125, 0.02178955078125, -0.21337890625, -1.0986328125, -0.10675048828125, -0.400390625, 0.95458984375, 1.7236328125, -0.337158203125, 0.05267333984375, 0.88720703125, 0.07745361328125, 0.09234619140625, 0.181884765625, 0.51171875, 0.591796875, 0.5087890625, 1.5556640625, 0.5341796875, -0.6943359375, 1.2119140625, -0.4072265625, -0.391357421875, 0.305908203125, -2.392578125, -0.2406005859375, 1.833984375, 0.1875, -1.154296875, -1.5400390625, 1.6318359375, -0.8486328125, -0.646484375, -1.24609375, 0.2423095703125, 0.0867919921875, -1.0029296875, 0.70654296875, 1.419921875, -2.5390625, -0.779296875, -0.335693359375, -0.7568359375, -0.370849609375, 0.044921875, 1.2119140625, -0.188720703125, -0.054046630859375, 0.61767578125, 1.7294921875, 1.111328125, 0.273193359375, 0.1669921875, 0.1981201171875, 1.048828125, -0.59326171875, 0.3583984375, 0.69970703125, 2.037109375, 2.4140625, 0.1549072265625, 0.77001953125, 1.2919921875, 2.216796875, -0.30078125, -0.0244598388671875, -0.1842041015625, -0.69775390625, 0.219482421875, 0.45849609375, 0.81689453125, 2.001953125, -0.849609375, 0.20068359375, 0.8662109375, 0.8916015625, 0.368408203125, 0.68603515625, -0.290771484375, 0.8056640625, 0.5625, -0.7880859375, -0.78369140625, -1.1044921875, 1.1494140625, 0.97314453125, 0.82666015625, 0.57177734375, 1.041015625, -1.1943359375, 0.58544921875, -0.81201171875, 1.861328125, -1.1689453125, 1.451171875, 0.419189453125, 1.138671875, -1.0908203125, -0.50390625, -0.0947265625, 0.40869140625, -1.447265625, -1.7841796875, -0.62353515625, -0.26708984375, -0.9150390625, 1.015625, -0.8447265625, 0.38037109375, -1.6728515625, 0.89404296875, 1.6826171875, -0.231201171875, -0.72314453125, -1.1298828125, 1.4423828125, 0.01509857177734375, -0.33447265625, -0.59033203125, -0.94775390625, 0.8359375, -0.255859375, -1.681640625, 1.0546875, -0.494384765625, 0.27294921875, -1.1396484375, 1.380859375, -0.43212890625, -0.0123443603515625, 1.0263671875, 0.0751953125, 0.343994140625, 0.11236572265625, 1.55859375, 0.334228515625, -1.3271484375, -0.65234375, -0.427978515625, 1.177734375, -0.045745849609375, 0.763671875, -0.7646484375, 0.441162109375, 0.865234375, 0.0697021484375, -0.03619384765625, -1.4482421875, 0.5302734375, 1.2978515625, -2.49609375, -2.123046875, 0.496826171875, 0.49462890625, -1.0576171875, -0.9267578125, -0.1876220703125, -0.174560546875, -0.72705078125, 2.158203125, 0.1380615234375, 0.9248046875, -1.546875, -1.2001953125, -0.380126953125, 0.11737060546875, 0.395751953125, 0.142822265625, 1.888671875, -1.6357421875, -1.3740234375, 0.765625, -0.982421875, 2.12890625, 0.82421875, -0.53759765625, 0.46044921875, -2.013671875, 1.576171875, -1.15625, 0.253173828125, 1.3193359375, 1.30078125, -0.3740234375, -1.6884765625, 0.95947265625, 0.0147857666015625, 1.09375, -0.375244140625, -0.9365234375, -0.98876953125, 0.4111328125, 0.9794921875, 0.018524169921875, 0.82421875, 0.366455078125, -0.0178680419921875, -0.1788330078125, 0.85888671875, 0.73681640625, 1.333984375, 1.3369140625, -1.8193359375, 0.81591796875, -0.87744140625, 1.5869140625, 1.0283203125, -1.3623046875, 0.26025390625, -0.7138671875, 0.90478515625, 0.7666015625, -0.78076171875, -0.239013671875, 0.80126953125, -1.259765625, -0.544921875, -0.2178955078125, 0.380859375, -0.61376953125, -1.595703125, 1.611328125, -0.5224609375, 2.048828125, -0.0316162109375, -0.2276611328125, 1.1845703125, 0.75537109375, 0.032562255859375, 0.68994140625, 0.5185546875, 0.822265625, -1.6142578125, 0.1746826171875, -0.3818359375, 0.72900390625, -1.1240234375, -0.71435546875, 1.2958984375, -1.201171875, -0.0280914306640625, -0.0418701171875, 1.7412109375, -0.95458984375, -0.144775390625, 0.14990234375, -1.365234375, 1.076171875, 1.6103515625, 0.71533203125, 0.5908203125, 1.4833984375, -1.2626953125, 0.430908203125, -0.75244140625, -1.83203125, -1.837890625, 1.3828125, 1.2958984375, -0.74609375, 1.7548828125, -1.0517578125, -0.07879638671875, -0.84033203125, -1.0634765625, -2.56640625, 0.9609375, -0.2255859375, -1.00390625, 0.459228515625, -0.353515625, -1.0263671875, -1.0751953125, -1.0791015625, -0.373046875, 0.5712890625, -1.037109375, 0.521484375, 0.314697265625, 0.7958984375, 0.0880126953125, 0.4013671875, 1.5732421875, 0.2310791015625, -0.43310546875, -0.3994140625, 0.2313232421875, 0.047760009765625, -2.33203125, 1.095703125, -0.038726806640625, -0.81982421875, 0.42333984375, -0.271240234375, -1.0732421875, -1.42578125, -0.07366943359375, 1.4482421875, 0.239013671875, -0.9580078125, -1.1962890625, -0.6982421875, 0.237060546875, 0.5888671875, 0.69384765625, -2.1328125, 1.1689453125, 0.0921630859375, -1.041015625, 1.484375, -0.2802734375, -1.6298828125, -2.037109375, -0.81787109375, -0.138916015625, -0.591796875, -1.015625, -0.8564453125, -1.16015625, 1.5732421875, -0.06329345703125, -0.1793212890625, -0.1287841796875, -0.71240234375, 0.6162109375, -0.79150390625, -1.0498046875, -0.29052734375, -1.8662109375, 0.1162109375, 0.2205810546875, 0.8857421875, -1.2744140625, 1.1298828125, 0.445556640625, -1.3642578125, 0.1978759765625, 1.892578125, -1.279296875, -0.7275390625, -0.271484375, -1.2314453125, 0.93212890625, 1.6181640625, -0.4091796875, 0.935546875, 0.55126953125, -0.75390625, -0.5322265625, 2.134765625, 0.2291259765625, 0.390380859375, -0.378662109375, 1.06640625, -1.0732421875, -1.123046875, 0.3193359375, 1.390625, 0.5546875, 0.407470703125, 1.6591796875, 0.5830078125, -0.58056640625, 0.9384765625, 0.59130859375, 2.078125, 0.2471923828125, -1.6083984375, 0.39990234375, 0.7841796875, 2.0859375, 0.5595703125, 0.436767578125, -0.6025390625, -1.3291015625, -0.2054443359375, -1.9736328125, -0.1759033203125, 1.3974609375, 0.55126953125, 0.7578125, 0.279541015625, -0.74609375, -0.358154296875, 0.363037109375, 1.48046875, -0.681640625, 0.2607421875, -1.0771484375, -0.8837890625, -0.75830078125, -0.271240234375, -0.004993438720703125, -0.90478515625, -0.8984375, -0.8642578125, 1.9619140625, 0.8154296875, 0.85498046875, -0.350341796875, -0.0146942138671875, -0.274658203125, -0.053466796875, -1.7490234375, -1.515625, 1.16796875, -0.08984375, 0.57373046875, 0.60498046875, -0.479248046875, 0.50537109375, 0.5341796875, 0.5234375, -0.450439453125, -2.779296875, 0.8154296875, 0.405029296875, -1.5869140625, 0.35009765625, 0.30908203125, 0.349365234375, 0.337646484375, 1.6630859375, -1.294921875, 0.63330078125, 0.7431640625, 0.07476806640625, 1.0078125, 0.2373046875, -0.1866455078125, -0.423095703125, -1.1181640625, 0.0167083740234375, 0.849609375, 0.1226806640625, -1.2119140625, -1.0576171875, -0.228515625, -1.142578125, 0.55126953125, 0.379638671875, 0.328369140625, -0.7080078125, 0.96142578125, -0.58544921875, 1.33203125, 2.93359375, 1.3994140625, 1.4970703125, -1.083984375, 2.630859375, 0.327880859375, 1.0283203125, 0.475341796875, -0.0243682861328125, -0.2374267578125, -0.483642578125, 0.49072265625, -1.3759765625, -0.1273193359375, -0.131103515625, -1.2333984375, 1.12890625, -0.5576171875, 0.50732421875, 0.35888671875, -0.83642578125, -1.2119140625, 0.900390625, 0.1492919921875, 0.96923828125, 2.13671875, 1.0009765625, -0.70166015625, -0.96142578125, 0.444580078125, -0.357421875, 0.28759765625, 1.4150390625, 0.0033721923828125, -0.220703125, 0.705078125, -0.58984375, 0.83154296875, 1.197265625, 0.294677734375, 0.05938720703125, 0.07659912109375, -0.6904296875, 0.363525390625, 0.400634765625, -1.7353515625, 1.2744140625, 0.280517578125, 0.0875244140625, 0.4208984375, -0.85302734375, 1.0166015625, 0.187744140625, -0.60400390625, 1.08984375, 0.0121917724609375, -0.165283203125, 0.049163818359375, 2.0234375, 0.962890625, -0.064208984375, 0.2257080078125, -1.8017578125, -0.5302734375, -0.288818359375, -1.2041015625, -0.72607421875, 0.392578125, -2.19140625, 0.190185546875, 0.70654296875, 0.9287109375, 1.9482421875, 1.7763671875, 1.888671875, 0.2335205078125, 0.26904296875, 0.0293426513671875, -1.0458984375, -0.140625, -2.263671875, 1.654296875, -0.625, 1.58984375, -0.0223846435546875, -0.05487060546875, -1.1484375, -1.244140625, 1.537109375, 0.82861328125, 0.462890625, 1.9072265625, -1.3662109375, -0.064453125, -0.791015625, -1.4384765625, 0.498046875, -0.9091796875, -0.259765625, -0.26123046875, 0.6328125, 0.16015625, -1.04296875, -2.259765625, 0.242431640625, 1.8095703125, -0.05816650390625, 1.1162109375, 1.271484375, -1.4033203125, 1.900390625, 1.2939453125, -0.59619140625, 0.4267578125, -0.1787109375, -0.423583984375, -1.484375, -1.328125, 2.1875, 0.2469482421875, -1.625, 1.51953125, 0.72802734375, 1.01171875, 1.2119140625, 0.2325439453125, 0.66943359375, 0.341552734375, 2.375, 0.525390625, 1.22265625, 0.274169921875, -1.203125, 0.0301513671875, -0.98046875, 0.45751953125, 0.37646484375, -0.49365234375, -1.6259765625, 0.552734375, -1.017578125, 2.23828125, 2.203125, -1.8271484375, -1.470703125, 0.533203125, -1.251953125, 2.205078125, 0.34033203125, -1.494140625, 0.54833984375, 0.59716796875, -0.1844482421875, 1.6787109375, 0.93798828125, -0.513671875, 0.9228515625, -0.0316162109375, 0.8291015625, 0.43017578125, 1.6474609375, 0.2252197265625, 0.134521484375, 0.873046875, 0.06512451171875, 0.337890625, -1.1201171875, -1.759765625, -1.44921875, 0.45947265625, 1.740234375, 2.220703125, -1.48046875, 0.275390625, -0.8564453125, -1.2177734375, 0.884765625, -0.9970703125, 0.12646484375, -1.1083984375, -0.84716796875, 0.1524658203125, 0.88720703125, 0.2244873046875, -2.142578125, 1.36328125, -0.14599609375, -1.40234375, 0.12066650390625, 1.595703125, 0.8916015625, -0.3583984375, 0.2481689453125, -0.97265625, -0.256103515625, -0.470703125, -1.576171875, 1.4677734375, 0.39990234375, 0.63232421875, -2.111328125, -1.7666015625, 0.391845703125, 0.5703125, 2.0, -0.626953125, -0.689453125, 1.193359375, 0.79296875, 0.09014892578125, -0.91796875, 0.85888671875, 0.68017578125, -0.861328125, 1.3984375, 1.7568359375, -0.0110015869140625, -1.248046875, -0.9462890625, -0.72509765625, 1.0400390625, -1.1181640625, -0.66796875, -0.49072265625, 0.14501953125, 0.42626953125, 0.62939453125, 0.034637451171875, 0.96337890625, 0.57861328125, 0.119873046875, -1.19140625, -0.45849609375, -0.6103515625, 0.90283203125, -0.5224609375, 0.69140625, -1.3173828125, -0.410400390625, 0.80908203125, 1.0478515625, -0.183837890625, -0.478759765625, 0.921875, 1.66015625, -0.8125, 0.1202392578125, -0.9970703125, 0.97607421875, -0.6611328125, 1.263671875, 1.6962890625, 0.196533203125, 0.3076171875, 1.42578125, 1.931640625, 0.76123046875, 0.3896484375, 0.10211181640625, -1.7412109375, -0.059967041015625, 0.1953125, -0.44873046875, -0.61181640625, 0.331298828125, 0.07330322265625, 0.04193115234375, -0.190185546875, 1.6806640625, -0.362060546875, 0.8203125, -0.1280517578125, 0.294677734375, 0.82958984375, 0.294921875, -0.84130859375, -0.032958984375, -1.32421875, 0.76220703125, -0.98291015625, 0.468505859375, 0.85400390625, 0.379638671875, -0.2086181640625, -0.216064453125, 0.93798828125, -0.370849609375, -0.79150390625, -0.003429412841796875, 0.951171875, -1.248046875, -0.92822265625, 0.11376953125, -0.3642578125, -1.087890625, 1.5078125, -0.47998046875, -0.98291015625, -0.5078125, -1.11328125, 0.61474609375, -0.407958984375, -0.47021484375, 0.36767578125, -0.396728515625, 0.85009765625, 0.771484375, -1.7685546875, -2.130859375, -0.90087890625, -1.3173828125, -0.67041015625, 0.94775390625, 1.0654296875, 0.61376953125, -0.1112060546875, -0.69482421875, 0.59326171875, -0.833984375, -2.15234375, 0.95849609375, -0.124267578125, 0.127197265625, 0.56494140625, -0.347412109375, -1.7099609375, 0.1270751953125, -0.374755859375, -0.2646484375, 1.880859375, -0.3203125, 1.669921875, -0.333984375, 0.033355712890625, -0.475341796875, -0.06573486328125, 0.6142578125, -0.7119140625, -0.505859375, 1.626953125, 1.056640625, 0.2490234375, -0.4794921875, 0.50537109375, -0.311767578125, -0.08392333984375, -0.419921875, -1.9931640625, 0.755859375, -1.7666015625, 0.73681640625, -0.7001953125, 2.119140625, -0.26123046875, 0.7548828125, 0.105712890625, -1.28125, -0.8642578125, 0.85986328125, -1.453125, -0.619140625, 0.66015625, -0.32470703125, 0.06768798828125, 0.298828125, -1.802734375, -1.6337890625, 0.48974609375, -0.173828125, 1.8125, 0.9697265625, -0.1693115234375, 1.212890625, -0.9228515625, -0.830078125, 0.3173828125, 1.9296875, -0.03558349609375, -0.05157470703125, -1.19140625, -0.5029296875, 2.4765625, -0.2880859375, -0.08367919921875, -1.84375, 0.56640625, 1.35546875, 1.021484375, 0.771484375, -0.285888671875, -0.2783203125, -0.42041015625, -1.8037109375, 0.6318359375, -1.9931640625, 1.1328125, 0.2186279296875, -0.7001953125, -0.0628662109375, -0.8642578125, 1.169921875, -0.5732421875, -1.015625, -1.7646484375, 0.5107421875, 1.740234375, -1.865234375, 1.1591796875, -0.309326171875, 0.485595703125, -0.73193359375, -0.39404296875, -0.035247802734375, 0.88623046875, 0.68359375, -0.76318359375, -1.40625, -0.427978515625, 0.52783203125, -0.174560546875, -0.313720703125, 0.4873046875, -0.7236328125, -1.37109375, -0.089599609375, -0.31982421875, 0.4326171875, 0.1627197265625, 1.6796875, -0.658203125, -2.63671875, -1.5400390625, 0.98486328125, -0.447021484375, 0.6982421875, -0.54736328125, -0.09307861328125, -0.373046875, 0.51708984375, 0.387939453125, 1.0595703125, 0.02264404296875, 0.359619140625, -0.494140625, -0.35693359375, -0.34375, 1.0859375, -0.93408203125, -0.375244140625, 0.284423828125, -0.356689453125, -0.00290679931640625, -0.923828125, 0.7890625, 0.6513671875, 0.052520751953125, -1.029296875, -0.90234375, 1.703125, 0.5947265625, 0.8369140625, -0.07470703125, 0.79443359375, 2.00390625, -0.83740234375, -0.28515625, -1.1552734375, 0.35107421875, -0.80126953125, 0.978515625, -0.00562286376953125, -0.80322265625, -2.056640625, -1.6591796875, 0.9111328125, 0.33447265625, 0.52099609375, -0.315673828125, -0.68798828125, 0.19970703125, 1.162109375, -1.2216796875, -0.25341796875, -1.130859375, -0.254638671875, -1.470703125, 0.634765625, -0.9208984375, 0.359130859375, 0.2109375, 0.826171875, -0.4345703125, -0.88427734375, 1.4248046875, -0.75634765625, -1.1123046875, 0.71728515625, 2.2734375, 2.662109375, 0.09088134765625, -0.74462890625, 0.76708984375, -1.7666015625, 0.6318359375, -1.04296875, 1.306640625, 0.79833984375, -1.712890625, 0.31689453125, 1.2412109375, 1.0849609375, -0.0789794921875, -0.35400390625, -1.486328125, 0.61279296875, -1.744140625, -1.5908203125, 0.16455078125, -1.1845703125, 0.8056640625, -0.360595703125, 1.21875, -0.626953125, -1.83203125, 0.17431640625, -0.25341796875, -0.64404296875, 0.82958984375, 0.931640625, 0.288818359375, -0.378173828125, -0.475830078125, 0.4267578125, 0.962890625, 1.1494140625, 2.646484375, -0.61865234375, 0.2149658203125, -1.9365234375, -0.295654296875, -0.06805419921875, 1.2353515625, -1.8359375, -0.97314453125, 0.2398681640625, -1.0751953125, 0.423095703125, 0.18115234375, 1.5224609375, 0.87353515625, -0.89306640625, 0.19091796875, -0.158935546875, -1.86328125, 0.006488800048828125, -0.364501953125, 1.3818359375, -1.158203125, -0.3046875, -0.346435546875, -0.330810546875, -1.765625, -0.370849609375, -1.0439453125, 0.00787353515625, -1.994140625, -0.646484375, -2.0546875, 0.314453125, 0.74755859375, -1.3662109375, -1.4775390625, -0.8193359375, 0.79736328125, -0.73681640625, 0.6083984375, 1.1005859375, -1.5048828125, -0.74609375, -0.468017578125, -0.93310546875, -0.37646484375, -0.053131103515625, 0.1746826171875, 0.179443359375, 0.2379150390625, -0.638671875, 0.98974609375, 0.345458984375, 1.435546875, 0.1514892578125, -0.96044921875, -0.11492919921875, 0.261474609375, -1.5341796875, 0.74609375, -0.6259765625, -1.529296875, 0.5693359375, -0.80908203125, 0.51953125, 0.332275390625, 0.47412109375, -0.68115234375, 0.615234375, -1.201171875, 0.439697265625, 1.5986328125, -0.57470703125, -1.7138671875, 1.052734375, -0.489990234375, -0.63525390625, 0.1865234375, -1.087890625, 0.95751953125, 0.6904296875, -1.201171875, 0.97705078125, 0.857421875, 0.197509765625, 0.441162109375, 0.92431640625, 1.40234375, 1.2724609375, -0.52978515625, -0.09063720703125, -0.478515625, -0.79052734375, 1.814453125, -0.72119140625, -0.12451171875, 1.55078125, -1.0380859375, -0.60009765625, 0.181884765625, -1.8017578125, 0.358154296875, -0.00885009765625, 1.115234375, 0.62060546875, 0.3681640625, 0.1578369140625, 0.82275390625, 1.3642578125, 1.0078125, 0.51708984375, 1.572265625, -0.414306640625, 1.3994140625, 0.2393798828125, -2.056640625, -1.0361328125, 1.5927734375, -0.060821533203125, -0.63134765625, -0.2412109375, -0.97216796875, -2.677734375, -0.71630859375, 1.0771484375, -0.0278472900390625, 0.28515625, -1.3515625, -1.1025390625, -1.09375, 0.8857421875, 0.775390625, 0.303466796875, -0.318359375, 0.283203125, -2.03515625, 2.544921875, -2.76953125, -0.6533203125, 0.1737060546875, -0.97216796875, -0.615234375, 0.05828857421875, -0.646484375, -0.65771484375, -0.283447265625, 0.144775390625, -0.1912841796875, 0.164306640625, -0.0273895263671875, -2.888671875, 1.59375, 0.1490478515625, 1.0966796875, 0.133056640625, -1.306640625, 1.1337890625, -1.5341796875, 0.464111328125, -1.9462890625, -0.63720703125, -0.92822265625, -1.6650390625, -0.479248046875, -0.66845703125, 1.3388671875, -0.14794921875, -0.67822265625, -0.214111328125, 1.326171875, 1.3115234375, -1.4970703125, 0.27734375, -0.51220703125, -1.203125, 0.56982421875, 0.0172882080078125, -1.7646484375, -1.2177734375, 0.333740234375, -1.26953125, -1.6142578125, -0.65966796875, 0.04364013671875, -0.29150390625, -1.591796875, 0.042236328125, 0.81884765625, -1.845703125, -0.88720703125, -0.75390625, -0.90478515625, -0.3828125, -0.407958984375, -1.2880859375, -0.96435546875, 0.0404052734375, 0.8115234375, 1.5478515625, 0.75927734375, -0.7998046875, -0.94775390625, 0.5126953125, -0.396728515625, 0.209228515625, -0.54443359375, -0.2919921875, -0.63525390625, -0.18310546875, 1.21484375, 0.169921875, 1.9765625, -2.546875, -1.1748046875, -1.630859375, 0.65625, -0.2607421875, 0.65185546875, -1.35546875, -0.9736328125, 0.65283203125, -0.982421875, 0.7119140625, 0.438720703125, 0.5361328125, -0.1787109375, -1.4140625, 0.22802734375, 0.50537109375, -1.1455078125, -0.53466796875, 0.9228515625, -2.41015625, 1.2158203125, 0.262451171875, -1.3564453125, -0.2100830078125, -0.0369873046875, 1.001953125, 0.1126708984375, -0.595703125, -1.3212890625, 1.669921875, -1.5029296875, 1.1611328125, -1.287109375, -0.4365234375, 0.1766357421875, 0.63818359375, -0.1597900390625, 0.49072265625, -0.41748046875, -0.61328125, -1.07421875, 0.72998046875, -0.318603515625, -0.1470947265625, -0.98095703125, -0.2078857421875, 1.2265625, 1.0244140625, 0.7548828125, 2.361328125, 0.5654296875, -0.172119140625, -2.38671875, -0.4384765625, -1.1708984375, -1.0625, -0.62255859375, 0.385498046875, -0.72412109375, 0.44189453125, -1.0673828125, 2.025390625, 0.87158203125, 0.64794921875, 0.081298828125, 0.12042236328125, 0.37744140625, -1.4140625, -0.5771484375, -0.330810546875, 1.431640625, -0.364501953125, -0.36865234375, 1.68359375, 0.110595703125, 1.201171875, 2.396484375, 0.90185546875, 0.1983642578125, 0.27734375, -0.283447265625, 0.91259765625, 0.10882568359375, -1.6171875, -1.357421875, -0.1751708984375, 0.83349609375, -1.2314453125, -0.6005859375, 0.44677734375, -0.0228118896484375, 1.6298828125, 0.96435546875, 2.0625, 0.250244140625, -0.020477294921875, 0.1702880859375, -0.52587890625, -0.7529296875, -1.94921875, -0.479248046875, 0.515625, -1.6416015625, -0.296142578125, -1.30078125, 0.79443359375, -1.0322265625, -0.66845703125, 1.1318359375, -0.68017578125, -0.99755859375, -1.0146484375, 0.006946563720703125, -0.71484375, -0.10198974609375, 2.201171875, 1.2607421875, 0.74267578125, 0.10430908203125, -0.89697265625, -0.5244140625, 0.042266845703125, 0.6708984375, -1.439453125, -3.26953125, 0.10626220703125, -0.1822509765625, -0.85498046875, -0.235107421875, 0.62841796875, 1.4326171875, -0.484130859375, -1.705078125, -0.79931640625, 1.3662109375, -0.767578125, -0.434326171875, 1.9287109375, 0.220703125, 2.640625, 0.324951171875, 0.324951171875, 1.3525390625, -0.179443359375, -1.1103515625, -0.8359375, -2.484375, 1.75390625, -2.06640625, 0.99609375, -0.451416015625, 1.26953125, 1.2880859375, -0.3154296875, 1.81640625, -1.2646484375, -0.43310546875, -2.6484375, 1.119140625, 0.88134765625, 1.921875, 0.703125, -0.63330078125, 1.5712890625, -1.0166015625, 0.10028076171875, 0.75, -0.65234375, -2.236328125, 1.236328125, 1.1044921875, 0.64111328125, 0.07745361328125, 0.7802734375, -0.1375732421875, 1.587890625, -0.48779296875, 1.560546875, -1.400390625, -1.490234375, 0.7412109375, -0.10467529296875, 0.333740234375, -1.0126953125, 0.52099609375, -0.6728515625, -0.258544921875, 0.5009765625, 0.68896484375, -0.35986328125, 1.107421875, 0.423583984375, -0.5234375, -0.6845703125, 0.2154541015625, -0.036590576171875, 0.92724609375, -0.422607421875, 0.443115234375, -0.0308685302734375, 0.6298828125, -0.703125, -0.219970703125, -0.342529296875, -0.8955078125, -0.78662109375, 1.7119140625, -0.39599609375, -0.055999755859375, 0.53369140625, -0.448486328125, 0.7421875, -0.78076171875, 1.8486328125, 1.2421875, -0.2418212890625, -0.09649658203125, 1.83203125, -0.9736328125, -0.00601959228515625, -1.0869140625, 1.53515625, -0.8193359375, 0.314208984375, 0.38720703125, -0.256103515625, -0.1318359375, -0.6416015625, 1.2021484375, 0.031158447265625, 0.90966796875, -0.420166015625, 0.061981201171875, 0.22607421875, -1.1044921875, -0.497314453125, -0.01438140869140625, -0.65380859375, -0.71630859375, -2.0078125, -0.131591796875, 0.1998291015625, 0.233154296875, 0.0185546875, -1.0810546875, -1.3994140625, -0.134033203125, 0.41552734375, 1.892578125, -0.537109375, -0.489990234375, -0.619140625, 2.0390625, -0.583984375, -0.2403564453125, 0.1944580078125, 0.08270263671875, -0.8359375, -1.056640625, -1.3603515625, -2.349609375, 1.916015625, 1.078125, 0.040863037109375, -0.367919921875, 0.69873046875, 0.54931640625, 0.432373046875, -0.74951171875, 0.728515625, 0.82373046875, -0.09234619140625, 0.5693359375, 0.1080322265625, 0.51220703125, -0.69482421875, 0.078369140625, -0.30615234375, 0.37744140625, 0.89599609375, 0.07354736328125, 0.414794921875, 0.10992431640625, -1.2763671875, -0.384765625, 0.361328125, 1.416015625, -0.4150390625, -0.21435546875, -0.68017578125, -0.68115234375, 0.78515625, -0.455322265625, 1.005859375, 0.41650390625, 0.55517578125, 0.98095703125, 1.3955078125, 1.033203125, -1.43359375, 0.517578125, -0.53369140625, -1.1240234375, 0.485107421875, -0.837890625, 0.2449951171875, 0.837890625, -1.177734375, 0.405517578125, -0.77294921875, -0.71435546875, 0.01016998291015625, 0.284912109375, -0.888671875, 0.0794677734375, -1.9873046875, -0.202880859375, -1.0908203125, 0.56396484375, 0.1824951171875, 1.0400390625, -0.5556640625, -0.09539794921875, 0.69921875, -1.0048828125, -0.83642578125, -1.3681640625, -0.54736328125, -0.26708984375, 0.51171875, 1.6728515625, 1.060546875, -0.358154296875, 0.30419921875, 0.63037109375, -0.171875, 1.63671875, -0.2744140625, 0.4453125, 0.1339111328125, 0.61572265625, -0.042266845703125, 0.1563720703125, -1.892578125, -0.187744140625, -0.1668701171875, -0.421875, -2.796875, -1.154296875, 1.470703125, 0.04205322265625, -2.484375, 0.96875, 0.27734375, -0.95166015625, 0.1912841796875, -0.6904296875, -0.759765625, -1.0107421875, -0.170166015625, -1.19921875, 1.212890625, -0.372314453125, -0.24755859375, 1.056640625, 0.433349609375, -1.4326171875, -0.0732421875, -1.123046875, 0.326416015625, 1.3095703125, 0.393798828125, 0.58935546875, 0.030364990234375, -0.75830078125, 1.552734375, 0.278076171875, 0.0723876953125, -0.3818359375, -0.6865234375, 0.86962890625, 1.9267578125, -0.37451171875, 0.180419921875, -1.2958984375, -1.1748046875, -1.7431640625, -1.796875, 1.974609375, -1.7734375, 0.48779296875, 0.44677734375, -0.0838623046875, 0.5849609375, 0.5390625, -1.533203125, 0.435302734375, -0.96142578125, 0.032196044921875, -1.046875, -0.9462890625, -1.0537109375, -0.89208984375, 0.72314453125, -0.11993408203125, -0.468017578125, -0.1365966796875, -1.4326171875, -0.4033203125, -1.306640625, 1.296875, 1.32421875, -0.67578125, 0.1522216796875, 0.231201171875, 0.285400390625, -1.5185546875, 0.974609375, 1.2666015625, 1.0517578125, -0.305908203125, -1.0419921875, 0.0859375, 1.3076171875, 1.3388671875, -0.1229248046875, -1.3154296875, -0.255126953125, 0.95458984375, 1.48828125, 0.62255859375, 0.346923828125, -0.970703125, 0.8916015625, 0.3935546875, 0.8935546875, -0.10150146484375, 0.09698486328125, -0.0994873046875, -1.6591796875, 0.0177764892578125, 0.4541015625, 0.405517578125, 1.12890625, -0.82177734375, 0.7431640625, 0.60693359375, 0.66162109375, -0.46435546875, -0.052276611328125, -1.294921875, -0.85546875, 1.3017578125, -0.5166015625, -0.4716796875, -0.65625, -0.5654296875, 0.429443359375, 0.64306640625, -0.72607421875, 0.78515625, -1.28515625, 1.2041015625, 1.9482421875, -0.91259765625, -1.984375, -1.65234375, 0.307861328125, 0.2032470703125, 0.599609375, 2.6875, 0.69775390625, 0.3828125, -0.60595703125, 1.3681640625, -0.5068359375, 0.60546875, 1.90234375, 0.09613037109375, -0.08642578125, -0.0867919921875, 0.76171875, -0.6796875, -0.1561279296875, 1.3349609375, 1.1416015625, 0.307373046875, -0.7919921875, -0.27685546875, -0.0273590087890625, -0.1854248046875, -0.74169921875, 1.22265625, -0.95556640625, 1.587890625, 0.1552734375, -0.61669921875, -0.338623046875, -0.216796875, -1.1953125, -0.054840087890625, -0.6728515625, -0.234619140625, 0.4677734375, 0.267578125, -0.1646728515625, -1.81640625, 0.2105712890625, -0.59228515625, -0.89404296875, -1.0908203125, 0.27587890625, 0.09796142578125, 0.1661376953125, -0.1259765625, 0.861328125, 0.6005859375, -0.146240234375, -0.7373046875, -0.720703125, -0.94140625, 1.275390625, 0.70556640625, 0.2548828125, 1.7431640625, -0.490966796875, 0.6201171875, -1.20703125, -0.65966796875, 2.431640625, 2.10546875, -1.294921875, 0.4921875, 1.2490234375, -0.165771484375, 0.33349609375, 0.59912109375, -2.484375, -0.52587890625, -0.32763671875, 1.3623046875, -0.298583984375, -0.0028591156005859375, 2.072265625, 0.70068359375, -0.1453857421875, -0.52001953125, -1.10546875, 1.5107421875, 1.638671875, -0.08062744140625, -0.115478515625, 0.43603515625, 0.5048828125, -0.2158203125, -0.638671875, -1.5078125, -1.9189453125, -1.4873046875, 0.16357421875, -1.1171875, -0.49169921875, 1.431640625, -0.4453125, 0.024658203125, -0.431396484375, -1.2548828125, -0.8134765625, -0.4423828125, 1.3408203125, 0.548828125, -0.1600341796875, -0.0301055908203125, 0.158447265625, 0.486572265625, -1.7099609375, 0.90625, -1.1484375, 0.53076171875, -0.75390625, -0.34765625, 0.218017578125, -0.265625, -0.1683349609375, -0.135009765625, 0.75341796875, -0.8720703125, 1.3203125, 0.61279296875, -0.159912109375, -0.92626953125, 0.07122802734375, -0.373291015625, -0.381103515625, 0.9423828125, 0.001129150390625, -0.9111328125, 1.1904296875, 0.50439453125, 1.09375, 1.203125, -0.265380859375, 0.65234375, -2.212890625, 0.8818359375, 0.42919921875, 0.059906005859375, 0.3427734375, 2.3359375, 0.1402587890625, 1.3662109375, -0.56640625, -0.1024169921875, -0.68701171875, -0.1453857421875, -0.65576171875, 0.297119140625, 0.3232421875, -0.17626953125, -1.3212890625, -0.58251953125, 1.021484375, 0.9072265625, -0.85888671875, -1.06640625, -0.1514892578125, -0.1861572265625, 2.01953125, 1.2470703125, 0.422607421875, -0.5986328125, 0.45703125, 0.783203125, -1.083984375, -0.328125, -1.724609375, 1.4619140625, 0.93701171875, 0.89404296875, 1.3515625, 0.16552734375, 2.029296875, -0.60888671875, -0.165283203125, -0.340087890625, -0.5634765625, 0.263671875, 0.167236328125, -0.83984375, 2.29296875, 0.343505859375, -0.475341796875, -0.8828125, -0.232421875, 1.3037109375, 0.005069732666015625, -0.9052734375, 0.1390380859375, -0.275146484375, -0.7548828125, -0.857421875, 0.89208984375, -0.475830078125, 0.051910400390625, 0.1494140625, -0.0848388671875, 0.38623046875, -0.66748046875, 0.81787109375, 1.1005859375, -0.440673828125, 0.1507568359375, 1.0634765625, 0.24169921875, 1.0126953125, -1.935546875, -1.896484375, 0.2919921875, 0.67138671875, 0.111572265625, 0.73583984375, 1.2568359375, 0.275634765625, 1.3671875, 0.253173828125, 0.68115234375, -0.37890625, -3.123046875, 1.3310546875, -0.86962890625, -0.74609375, -0.79443359375, 1.1728515625, 0.432373046875, -1.47265625, -0.5732421875, -1.9453125, 0.12139892578125, -1.189453125, -2.1171875, 2.390625, -0.3310546875, 0.30517578125, -1.66015625, -0.62158203125, -1.51171875, 0.467529296875, 0.58837890625, 0.71533203125, -0.75439453125, -0.83642578125, 0.252197265625, -0.6474609375, 0.82275390625, -0.47802734375, 1.326171875, -0.36279296875, -1.1337890625, -0.307861328125, 1.2763671875, -1.189453125, -0.414794921875, -0.1878662109375, 0.289794921875, 0.06768798828125, -1.021484375, -1.693359375, 0.58740234375, -1.0869140625, 0.65869140625, -0.89404296875, 1.08203125, -0.51025390625, 0.186767578125, -0.39697265625, 0.81005859375, 0.1865234375, 0.1402587890625, 0.454345703125, -0.341064453125, 1.3125, 0.41357421875, 0.393798828125, -0.4931640625, 0.6484375, -1.2099609375, -0.1650390625, 1.6591796875, 0.5986328125, 0.1536865234375, 0.8203125, 0.104248046875, -0.144287109375, 1.4267578125, -0.5107421875, 0.8349609375, 1.4130859375, -0.56103515625, 0.0498046875, 1.3330078125, -0.99658203125, 0.91064453125, 0.1160888671875, -0.6201171875, -0.42236328125, 0.1263427734375, 0.08697509765625, -0.488525390625, 0.1829833984375, 0.47705078125, -0.3388671875, 0.496826171875, -0.69140625, -0.8955078125, -0.493408203125, 0.7109375, -0.139892578125, -0.28759765625, 0.2159423828125, 0.11785888671875, -0.341552734375, -0.31591796875, -1.400390625, -0.1370849609375, -0.8037109375, -0.2020263671875, 0.73583984375, 1.365234375, 2.212890625, -0.2362060546875, -0.313720703125, 0.83056640625, -0.8515625, -1.869140625, 0.69091796875, 0.826171875, -0.52294921875, -0.9091796875, -1.08203125, 1.0537109375, -0.509765625, -0.2392578125, -0.54052734375, 0.1427001953125, -0.599609375, -1.599609375, 1.0419921875, 0.443603515625, -0.278564453125, 0.1656494140625, -0.403564453125, 0.6240234375, 0.264892578125, -1.263671875, -0.5087890625, 0.873046875, 0.75634765625, -0.6640625, 0.409912109375, -1.2158203125, 0.372802734375, -0.433837890625, 1.0390625, 0.08831787109375, -0.49169921875, 1.5439453125, -0.4912109375, -1.78125, 1.921875, -2.12109375, -0.71435546875, -1.6279296875, 1.7216796875, 0.63916015625, -0.70556640625, 1.3740234375, 1.400390625, -0.064208984375, -0.443359375, -1.5712890625, 0.35400390625, -0.031890869140625, -0.389404296875, -1.373046875, -0.748046875, 0.2607421875, 1.537109375, 0.72705078125, 1.349609375, 0.8486328125, 0.165771484375, -1.083984375, -0.185791015625, -1.701171875, 0.1939697265625, -0.050994873046875, -0.491943359375, -1.3876953125, 0.53955078125, 1.0048828125, 0.8115234375, 0.72607421875, 0.51171875, 0.87841796875, -1.6181640625, -0.040313720703125, -1.10546875, 0.1790771484375, 0.08905029296875, -0.53466796875, -0.07904052734375, 0.2044677734375, -0.96484375, 0.955078125, 0.286865234375, 0.297119140625, -0.79443359375, 0.282958984375, 0.442138671875, -0.783203125, 0.9365234375, 0.56640625, 2.68359375, -1.3037109375, -1.640625, 1.3466796875, 0.045562744140625, -1.6513671875, 0.81201171875, 0.134033203125, 1.357421875, -1.1259765625, -0.888671875, -0.331787109375, 1.3076171875, 0.84521484375, 0.40087890625, -2.02734375, -1.0009765625, -2.6328125, -0.78466796875, -0.7255859375, -0.9345703125, 1.787109375, -2.38671875, 0.9775390625, 0.59716796875, 0.56591796875, -0.673828125, 0.87451171875, -0.53173828125, -1.1552734375, 0.3857421875, 1.294921875, -0.342041015625, 0.10601806640625, 1.330078125, 1.2333984375, -0.2366943359375, 1.0703125, 0.306884765625, 0.7802734375, -0.4423828125, 0.263671875, 1.7451171875, 0.8505859375, 0.1375732421875, 1.3857421875, -0.509765625, -2.111328125, -1.00390625, -1.0908203125, 0.78466796875, 2.404296875, -0.9716796875, -0.315673828125, 1.744140625, -1.3935546875, 0.490478515625, -0.33544921875, -0.47119140625, 1.55078125, 0.62939453125, -0.1268310546875, -0.53857421875, 1.109375, -0.8583984375, -0.50146484375, 0.330322265625, -1.2783203125, -0.9052734375, 1.9208984375, 0.69677734375, 2.078125, 0.319580078125, 1.7373046875, 0.184326171875, 1.1171875, -0.47802734375, 0.32275390625, -2.130859375, 1.04296875, -0.1390380859375, 0.68017578125, 1.11328125, -0.0614013671875, -0.81103515625, 2.29296875, -0.63720703125, -0.30517578125, -0.285400390625, -0.84326171875, -1.7734375, 1.4775390625, 0.79736328125, -2.12890625, -1.1396484375, 0.362060546875, 1.1181640625, 2.185546875, -1.73828125, 1.2119140625, 2.197265625, 1.197265625, -0.5771484375, 0.41943359375, -0.81298828125, 1.2392578125, -0.47509765625, 0.379150390625, -0.8310546875, -0.58642578125, 0.267333984375, -2.955078125, 0.64453125, -0.71240234375, -0.763671875, -0.74951171875, 2.392578125, 0.69091796875, 1.7568359375, -1.4482421875, 0.96923828125, 0.92333984375, -0.083984375, 1.2919921875, 1.166015625, -0.0228729248046875, -0.418212890625, 1.681640625, -0.2142333984375, 1.4580078125, 0.55517578125, -2.466796875, -2.224609375, -0.5751953125, 0.97900390625, -2.001953125, 1.08203125, 2.078125, 0.67041015625, 0.833984375, -1.5830078125, -2.10546875, 1.4189453125, 2.037109375, 0.11846923828125, -1.009765625, -1.0087890625, -1.1064453125, -0.5166015625, -0.84228515625, 0.3349609375, -0.52392578125, 0.6552734375, 0.258544921875, 1.1435546875, 0.2841796875, 0.44287109375, -1.2236328125, -0.5390625, -0.297607421875, -0.07952880859375, 1.076171875, 0.33349609375, -1.1494140625, -0.53369140625, -0.26806640625, -0.88330078125, -0.00812530517578125, 0.336181640625, -1.0, 0.4755859375, -0.038970947265625, -1.130859375, 0.36572265625, -0.7900390625, 0.89306640625, -0.95556640625, -1.1416015625, -1.40625, -1.60546875, -0.30322265625, -0.405029296875, -1.60546875, -0.65087890625, -0.293701171875, -1.9501953125, 0.6767578125, -0.296875, -0.5625, 1.302734375, -0.8466796875, -0.362060546875, 1.3056640625, -2.009765625, -0.5556640625, 0.293212890625, -0.9189453125, -0.84326171875, -0.04156494140625, 0.69189453125, -1.060546875, -0.60791015625, -1.03515625, -0.34765625, -0.888671875, -0.5439453125, 0.138916015625, -1.3544921875, -2.138671875, 0.232421875, -1.1298828125, -0.380859375, -0.689453125, -0.127685546875, 0.61083984375, 1.666015625, 1.8271484375, -0.301025390625, 0.92138671875, 2.19140625, 0.1561279296875, 1.4443359375, 0.2252197265625, 1.6533203125, 0.1319580078125, -0.4697265625, -1.267578125, -1.197265625, 0.2032470703125, -1.490234375, 0.5556640625, -0.429443359375, 1.048828125, -0.62939453125, 1.1689453125, 0.7978515625, -0.276123046875, 1.236328125, -0.6630859375, 1.5361328125, 0.408935546875, 0.115478515625, -0.303955078125, 1.5478515625, -0.7314453125, -1.380859375, 0.493896484375, -0.298583984375, -1.458984375, 1.60546875, -0.23046875, 0.0266571044921875, 1.298828125, 0.034210205078125, 1.2294921875, -1.5009765625, -0.79736328125, -1.568359375, 0.45947265625, -0.978515625, -0.10650634765625, -0.69140625, -0.3984375, 0.900390625, 1.427734375, -0.58935546875, 1.2548828125, 0.55810546875, 0.2294921875, -1.0390625, 0.98681640625, -0.185791015625, 2.158203125, -1.44921875, -1.7724609375, -2.353515625, 0.755859375, 0.460693359375, 0.71826171875, -0.80078125, 1.947265625, 0.6162109375, -0.34521484375, 1.9150390625, 0.71728515625, 0.317626953125, -1.4384765625, 1.609375, 0.57373046875, 1.17578125, -1.1220703125, 0.71728515625, 1.1923828125, -0.476806640625, -0.42919921875, 0.1734619140625, -0.44970703125, 0.91162109375, -0.0218658447265625, 0.11248779296875, 1.5302734375, -0.64453125, -0.55322265625, 2.537109375, 1.9140625, -1.3662109375, -1.125, -0.114990234375, -0.279296875, 1.2392578125, -0.4833984375, 1.478515625, 0.2744140625, -0.419921875, 0.27392578125, -0.6962890625, -1.66015625, 0.478759765625, -1.41796875, -0.75634765625, -1.25390625, -1.544921875, -0.5009765625, -0.476318359375, -0.438720703125, 0.333740234375, -0.32177734375, 0.708984375, -0.288330078125, 0.7119140625, -1.712890625, 0.7607421875, -1.1787109375, -0.468017578125, 0.66943359375, -1.0615234375, 0.0631103515625, -0.50146484375, -1.0771484375, -0.576171875, -1.4013671875, 2.107421875, -0.65673828125, 0.239013671875, -0.386962890625, 0.88427734375, 0.76904296875, 0.5419921875, 2.103515625, 0.67578125, 0.451416015625, 0.0684814453125, -0.259033203125, 0.06640625, 1.3798828125, 0.71142578125, 0.8994140625, 1.400390625, 1.458984375, -1.3857421875, 1.962890625, 1.9130859375, -0.40625, -0.4697265625, -0.86669921875, 1.5400390625, 0.82861328125, 0.061920166015625, 0.195068359375, -0.591796875, 0.07464599609375, -0.484375, 0.006290435791015625, -1.033203125, -2.177734375, -0.6171875, 0.59423828125, -1.876953125, 1.7939453125, 0.6640625, -0.35400390625, -1.31640625, -1.86328125, 0.7587890625, 0.2225341796875, 0.6240234375, 1.5009765625, -1.5068359375, 1.3193359375, -1.2080078125, -1.228515625, -0.6474609375, 1.1416015625, -0.3291015625, -0.30078125, 1.333984375, -0.220703125, -0.33740234375, 0.74560546875, -0.049591064453125, 1.2578125, -0.69775390625, -1.212890625, -2.3203125, 0.307373046875, 0.08721923828125, 0.12109375, 0.396484375, -0.0364990234375, -1.80078125, 0.6279296875, 0.103271484375, 1.3603515625, -1.9833984375, -0.2279052734375, -0.385009765625, 1.74609375, -0.0892333984375, -1.2587890625, -0.215576171875, -0.203125, 1.1669921875, -0.6005859375, -1.947265625, -0.9296875, 0.56689453125, -0.95068359375, 0.29296875, -0.9306640625, -1.3798828125, -0.1268310546875, 2.474609375, -1.052734375, 0.619140625, -0.64794921875, 0.06298828125, 1.5078125, -0.0650634765625, 2.228515625, -0.67041015625, -0.11846923828125, -1.7197265625, 0.5400390625, -0.431396484375, 0.8984375, -0.1517333984375, -1.1953125, 0.740234375, -0.8818359375, -0.62255859375, 1.5859375, 1.33203125, -0.0384521484375, -1.7685546875, 0.2301025390625, 0.1414794921875, 0.8310546875, 0.481689453125, 1.7294921875, -0.45458984375, 0.3623046875, -1.154296875, 1.1748046875, 0.229736328125, 1.013671875, -1.2392578125, 0.56640625, -0.58251953125, -0.9580078125, -1.4287109375, 0.71484375, 0.401123046875, -0.496826171875, -1.0400390625, 0.50390625, -2.259765625, 0.60205078125, 0.20166015625, 0.04803466796875, 0.90625, 0.88134765625, 0.32470703125, -0.376708984375, -1.0732421875, 0.279296875, 0.45849609375, 0.2203369140625, 1.7734375, -1.65625, 0.10552978515625, -1.84765625, 0.1185302734375, -0.260986328125, -0.6865234375, 1.0068359375, -0.23876953125, -0.25390625, -0.90283203125, 0.10498046875, -1.8056640625, 0.5634765625, 0.4072265625, 0.365234375, 0.61572265625, 0.5361328125, 2.087890625, -0.406494140625, 1.2158203125, -0.440185546875, -0.279541015625, -0.90478515625, -0.230224609375, 0.1435546875, -2.3828125, 0.65380859375, 0.0521240234375, 1.6572265625, -1.0556640625, -0.324462890625, 0.87841796875, -0.6171875, 0.58935546875, 0.6396484375, -0.560546875, -1.8955078125, -1.5595703125, -0.126220703125, -0.6416015625, 0.72509765625, -0.051788330078125, -0.35888671875, -0.462890625, 0.84423828125, 0.923828125, 1.0869140625, 0.56396484375, -0.9189453125, 0.8037109375, -0.377685546875, 0.9189453125, -0.77001953125, 1.064453125, 0.55517578125, 0.61376953125, -0.9921875, -0.260498046875, -0.356201171875, 0.89599609375, 0.441650390625, 0.212646484375, -0.28125, -0.75244140625, -0.82470703125, -1.4296875, 1.0498046875, 1.8447265625, -0.8701171875, -0.0223236083984375, -0.348876953125, -0.08966064453125, 0.037017822265625, -0.3125, 0.383544921875, 1.2275390625, -0.78125, -1.6337890625, -0.53125, 0.99365234375, 1.6025390625, 0.081787109375, 0.5927734375, 1.7490234375, -0.82421875, -0.051361083984375, 1.1708984375, 0.587890625, 2.275390625, 0.7578125, -0.8603515625, 1.14453125, 0.75341796875, 1.369140625, -1.6416015625, 0.7890625, -0.7587890625, 1.283203125, 0.3359375, -0.327880859375, 0.2890625, 0.169189453125, -1.13671875, 1.34765625, 1.4306640625, -0.223388671875, 0.11090087890625, 0.202880859375, -0.428466796875, 2.453125, -0.38671875, -0.890625, -0.0178375244140625, -0.53759765625, -1.44140625, 0.278076171875, -0.76318359375, -1.6669921875, 1.33203125, -0.63623046875, 0.26171875, 1.8623046875, -1.224609375, 0.9375, -0.428955078125, -0.2076416015625, 1.3642578125, 1.0224609375, 0.5673828125, -0.48876953125, -1.4443359375, 0.8916015625, -1.400390625, 0.98779296875, 0.7080078125, -0.62109375, -2.228515625, -1.39453125, -1.572265625, -0.27197265625, 1.4267578125, 1.271484375, -1.0048828125, 1.501953125, -0.93310546875, 0.479248046875, 0.27392578125, 0.01078033447265625, 0.467041015625, -0.080322265625, -0.041473388671875, -0.1690673828125, -0.4375, -1.1552734375, -0.865234375, 0.222412109375, 0.7314453125, -0.40869140625, 0.09478759765625, 0.332275390625, 0.361328125, -0.826171875, 0.18017578125, -0.06494140625, 1.59765625, 1.0771484375, -1.3095703125, -0.53955078125, 1.1650390625, -0.1392822265625, -0.2264404296875, 0.237548828125, 1.1181640625, 0.93896484375, 1.388671875, 0.81201171875, -0.58740234375, 0.450927734375, 0.16357421875, 0.5205078125, -0.69384765625, 0.437744140625, -1.32421875, 0.873046875, -1.12109375, -0.261962890625, 1.79296875, 0.73486328125, 0.60888671875, -2.35546875, 0.397216796875, 1.0634765625, -0.1251220703125, 0.8603515625, -0.48388671875, 0.10430908203125, 0.351806640625, -1.8310546875, 0.1390380859375, -0.90234375, 0.36669921875, -0.399169921875, 0.53955078125, 1.2998046875, -1.41015625, 1.85546875, 0.251708984375, 0.033477783203125, 0.0360107421875, 0.107666015625, -0.251953125, -1.154296875, 1.89453125, 0.703125, -1.966796875, -0.1583251953125, -1.349609375, 0.55517578125, -1.896484375, 0.292724609375, 0.61376953125, 0.312255859375, 1.1806640625, -0.79833984375, 0.05487060546875, 0.5810546875, 0.9833984375, 2.091796875, 1.6318359375, 2.205078125, -0.6953125, -1.2294921875, -0.30712890625, -0.73095703125, -1.03125, 1.3935546875, 0.19384765625, -0.00463104248046875, 1.0673828125, 0.955078125, 0.2022705078125, 0.153076171875, 0.1710205078125, -0.07244873046875, 0.82275390625, 0.58203125, -0.443115234375, 0.982421875, -1.1767578125, 1.166015625, 2.044921875, 0.451171875, 0.1256103515625, 2.001953125, 0.76708984375, 0.35693359375, 0.9169921875, -0.96240234375, 0.662109375, 1.640625, -0.050323486328125, -2.3515625, 0.0269317626953125, -0.4951171875, -0.3271484375, 0.2039794921875, -0.2626953125, 0.56396484375, -1.4443359375, 0.31494140625, 0.0546875, -0.14599609375, -1.265625, 1.3408203125, -0.599609375, -0.7373046875, -0.46435546875, -1.3095703125, 0.57080078125, -0.10906982421875, -0.88720703125, -0.58642578125, -1.994140625, -0.22021484375, -0.1910400390625, 1.1650390625, -0.58837890625, -0.3125, 1.5078125, -0.0286407470703125, 0.055755615234375, -0.469482421875, 1.4931640625, 1.6806640625, 1.625, 0.428955078125, 1.14453125, 1.2275390625, 0.415283203125, 0.68359375, -1.109375, -0.452880859375, 0.443115234375, -0.453369140625, 0.28466796875, 0.1878662109375, 0.8671875, 0.25146484375, 0.556640625, -2.134765625, -0.61767578125, -0.29736328125, -0.51904296875, 0.2271728515625, -0.62109375, 1.12890625, 1.095703125, -0.63525390625, 0.89404296875, -1.3017578125, -0.36083984375, 0.259521484375, -0.21728515625, 0.0440673828125, 1.744140625, -1.3203125, 0.017059326171875, -0.72314453125, 1.0048828125, 0.369384765625, 0.20068359375, 0.192138671875, 0.032379150390625, 0.397705078125, 1.3505859375, -2.14453125, 0.374755859375, -0.0001379251480102539, 0.2208251953125, 0.136962890625, -1.72265625, 0.55712890625, -1.13671875, -0.38330078125, 0.355224609375, 0.47705078125, 0.1650390625, -0.1923828125, 0.765625, -1.087890625, 0.06170654296875, -0.97802734375, 0.416748046875, -0.10089111328125, 0.1878662109375, -0.68359375, -0.2489013671875, -0.396484375, -0.873046875, 0.7314453125, 2.02734375, 1.7119140625, 0.5458984375, 1.07421875, 0.9541015625, -2.232421875, -0.83837890625, 0.1875, -0.2161865234375, 0.056121826171875, 0.53955078125, 0.626953125, -0.42724609375, -0.307373046875, 0.7451171875, 1.2939453125, 0.2364501953125, -0.61181640625, -0.06597900390625, -1.2890625, -1.62890625, -1.1787109375, -1.4375, -0.3291015625, -0.7548828125, 0.463623046875, -0.25341796875, 0.08184814453125, -1.8095703125, 0.2900390625, -0.8046875, -1.814453125, -0.32568359375, -0.453125, -0.59130859375, -2.099609375, 0.491455078125, 0.195556640625, 0.2099609375, 0.97314453125, 1.7412109375, 0.908203125, -0.67919921875, 0.147216796875, -1.353515625, -0.427978515625, 0.2225341796875, 0.49365234375, -0.85888671875, 0.87548828125, -0.406005859375, -1.4560546875, -0.71875, -0.262939453125, 0.1627197265625, -2.208984375, 1.7236328125, 0.87548828125, -0.8974609375, 0.5107421875, 0.07843017578125, -0.9384765625, -0.447509765625, -0.8505859375, -0.85791015625, 1.7041015625, -0.82666015625, 2.80078125, -0.329345703125, 1.99609375, -1.1318359375, 0.796875, -0.429931640625, 0.0537109375, -0.666015625, -0.286376953125, -0.7333984375, 1.2216796875, 0.759765625, 0.62109375, -0.89501953125, -1.4765625, 0.51513671875, 0.96533203125, -1.51953125, -0.188232421875, -0.66015625, -0.192626953125, 0.1402587890625, -0.477783203125, -0.82568359375, 0.32763671875, 0.60546875, -1.890625, 0.296875, 0.53662109375, 1.21875, 0.1903076171875, -0.9306640625, -0.765625, -2.14453125, -0.0185699462890625, 0.75732421875, 0.84228515625, -0.8349609375, 0.2064208984375, 0.429931640625, 0.336181640625, -2.115234375, 1.26953125, 0.30859375, 0.5380859375, -0.1976318359375, -0.53564453125, 0.042572021484375, 0.00848388671875, 1.3271484375, 0.1722412109375, 0.71337890625, -2.091796875, 0.9267578125, 0.97119140625, -1.4736328125, -1.365234375, 0.1995849609375, 1.9580078125, 0.29736328125, 0.75537109375, -0.2259521484375, 0.7236328125, 1.2666015625, 2.20703125, -2.404296875, 1.4541015625, 0.65380859375, 0.411376953125, -0.68212890625, 0.69775390625, -2.6640625, -1.2978515625, 1.2841796875, -0.1466064453125, 1.0185546875, -0.1866455078125, 0.53076171875, 0.10235595703125, -0.484619140625, 1.9140625, -0.90478515625, -1.4990234375, 1.609375, 1.478515625, 0.2130126953125, 0.425048828125, 0.744140625, 0.5517578125, -0.96435546875, -0.2105712890625, -0.306640625, 0.80859375, -0.537109375, -0.95556640625, 0.74609375, 0.473388671875, -0.37646484375, -1.126953125, -0.343505859375, 0.63916015625, 0.86767578125, -1.14453125, -0.51708984375, 0.79931640625, 1.4736328125, 1.65234375, 0.41748046875, -0.369873046875, -0.87158203125, 0.96533203125, -0.1641845703125, 0.054290771484375, 1.373046875, 0.740234375, 0.76123046875, 0.67138671875, -1.1474609375, -1.3671875, -1.603515625, -0.366943359375, 0.30078125, 0.3017578125, -0.356201171875, -0.9296875, 0.548828125, 0.360107421875, 0.3720703125, -1.4755859375, 1.3154296875, 1.01171875, -0.5849609375, -0.2225341796875, -0.75341796875, 0.6923828125, 1.94140625, 0.437255859375, -0.95361328125, 0.7158203125, -0.4140625, 1.5888671875, 1.22265625, 1.26953125, -0.91162109375, -2.275390625, 1.583984375, 0.81396484375, 1.3046875, -1.234375, -0.38525390625, -1.2138671875, -1.5263671875, -2.51953125, -1.5517578125, -0.5654296875, -1.7607421875, 0.5126953125, -1.3212890625, 0.335693359375, 0.311767578125, 0.9990234375, -0.217041015625, -0.266845703125, 1.4150390625, -1.2021484375, -0.412109375, -0.415771484375, 0.382080078125, 1.1923828125, 0.2064208984375, 0.94091796875, 0.05029296875, -1.056640625, 2.3671875, 0.319580078125, 0.65625, -0.54296875, 0.53759765625, 1.76171875, 0.712890625, -0.59375, -1.05078125, 0.24560546875, 1.1875, -0.275146484375, -0.3876953125, 1.0087890625, -1.029296875, 0.76416015625, 0.4892578125, 0.71435546875, -0.56005859375, 0.53076171875, -1.5654296875, 0.72705078125, -0.09173583984375, -0.135498046875, -0.8896484375, 0.77978515625, -0.1600341796875, 0.97705078125, -0.81689453125, 1.0478515625, -0.53515625, -0.91064453125, 1.3271484375, -0.78466796875, 0.708984375, 1.5283203125, -1.2080078125, 1.12890625, -0.491455078125, -1.1845703125, 1.9462890625, -1.166015625, -0.68994140625, -0.2127685546875, -2.3046875, 0.31884765625, 2.37109375, -0.9140625, 1.3271484375, -1.05078125, -0.3173828125, -1.5927734375, 0.9189453125, -0.166748046875, 2.29296875, 0.794921875, 0.32861328125, -1.16015625, -0.392822265625, -0.64990234375, -1.927734375, -0.6279296875, 2.830078125, -0.908203125, 0.06683349609375, 1.31640625, 0.384033203125, 1.681640625, -2.185546875, -0.315673828125, 1.302734375, -1.16796875, -0.0100860595703125, -0.98291015625, 1.0546875, 0.86767578125, -0.58447265625, 0.68896484375, 0.025146484375, 0.59326171875, -1.4189453125, 0.609375, 0.0284576416015625, 0.49609375, 0.53564453125, 0.62890625, 1.88671875, 0.583984375, 0.33642578125, 0.35400390625, 0.38623046875, 0.5625, 1.0, 0.211669921875, -1.7880859375, -0.8662109375, -1.21875, 2.001953125, -0.82080078125, -0.80517578125, 2.171875, 0.267333984375, -0.3662109375, 0.8271484375, -0.1383056640625, 0.1795654296875, 2.26171875, -1.0361328125, 0.775390625, 0.1197509765625, -0.1607666015625, 1.2890625, 0.0173492431640625, -1.39453125, 1.0048828125, 0.249755859375, 1.4609375, 0.27490234375, 0.2137451171875, 1.1357421875, 1.4521484375, -0.666015625, 0.016754150390625, 0.03411865234375, 0.5703125, -0.235595703125, 1.0791015625, 0.642578125, 0.2415771484375, 0.8525390625, 0.5478515625, -0.8369140625, 0.51171875, 0.060333251953125, 0.80419921875, 0.8671875, -0.47314453125, -0.358642578125, 0.362060546875, -1.6982421875, 2.359375, 0.53955078125, -0.8291015625, -1.63671875, -1.1513671875, 2.099609375, -0.71142578125, 0.445068359375, -0.57568359375, 1.490234375, 0.640625, 1.17578125, -0.4345703125, 1.271484375, -1.228515625, -1.25, 0.76416015625, -0.52294921875, -0.32373046875, -0.8642578125, 0.509765625, 1.177734375, 0.62255859375, -0.425537109375, -1.9091796875, -0.457763671875, -0.654296875, 0.0916748046875, 0.203369140625, -1.134765625, -0.06494140625, 0.237060546875, 1.0546875, 0.54150390625, 0.9208984375, -1.14453125, -0.295654296875, -1.384765625, 0.0008420944213867188, -0.05474853515625, 0.8125, -0.60986328125, 0.232421875, -0.418701171875, 0.193359375, 0.430908203125, 0.1593017578125, 0.755859375, -0.1761474609375, -0.48828125, -0.6572265625, -0.76220703125, 1.4765625, -1.73828125, -1.5068359375, 0.64892578125, -1.1591796875, 0.197998046875, 1.2421875, 0.65283203125, 1.6171875, 0.060272216796875, 0.279296875, 0.4921875, -1.5322265625, 0.369140625, 1.0517578125, -1.798828125, 0.63916015625, 0.01522064208984375, -1.4521484375, 0.369873046875, 0.93310546875, 0.381591796875, -0.6083984375, -0.167236328125, -1.38671875, -1.095703125, -1.0078125, -0.282958984375, -1.76171875, -1.0751953125, -1.607421875, -2.2109375, -1.1337890625, -0.052978515625, 0.771484375, 0.319091796875, -0.357666015625, -0.71923828125, -0.83935546875, -0.272216796875, 0.5615234375, -1.0224609375, -0.90673828125, -1.984375, -1.9033203125, -0.330810546875, 1.279296875, 0.78271484375, 0.162109375, -0.79541015625, 1.0478515625, 2.3671875, 1.0087890625, 0.78466796875, 0.9375, 0.4970703125, -0.59521484375, 1.236328125, -1.658203125, 0.492431640625, -0.68310546875, -1.1025390625, -0.125244140625, -0.6826171875, 1.19921875, -1.7880859375, -0.006130218505859375, -1.7392578125, -0.72802734375, -0.058197021484375, -0.42822265625, -0.33984375, 1.181640625, -1.0859375, -0.09661865234375, 1.5771484375, 0.08795166015625, 0.329833984375, 0.2471923828125, -1.2080078125, 1.263671875, -0.240234375, 1.7509765625, -0.2215576171875, 0.5166015625, 1.2509765625, -1.69921875, -0.248291015625, 0.377685546875, -1.4755859375, 0.83642578125, 0.430908203125, -0.74853515625, -0.51513671875, -1.1298828125, 1.1298828125, 1.6552734375, 1.5673828125, 0.1234130859375, -1.392578125, 0.0933837890625, -1.1328125, -0.9169921875, 0.0816650390625, -2.998046875, -2.0546875, 0.6328125, 0.39990234375, -0.0294647216796875, -0.1849365234375, -0.62451171875, -0.64208984375, 0.0213165283203125, -0.08935546875, -0.1446533203125, -0.477294921875, -0.08087158203125, -1.1044921875, 1.9306640625, 0.0013875961303710938, 0.75537109375, 0.5107421875, -0.970703125, 0.3671875, -0.36279296875, -0.26513671875, 0.227294921875, 0.7646484375, 1.2275390625, 2.123046875, 0.51806640625, -0.085205078125, -0.5712890625, 0.10150146484375, 0.51953125, 1.2333984375, -0.67138671875, 1.7451171875, -1.068359375, -1.2783203125, 0.2236328125, -0.77099609375, -0.35888671875, 0.80712890625, -0.71240234375, -0.436767578125, 0.66943359375, -1.294921875, 2.15625, 2.30078125, -0.734375, 1.7353515625, -0.86181640625, 0.317138671875, 1.30859375, -1.154296875, -1.8984375, 0.1331787109375, 0.671875, 1.19140625, 0.85107421875, -0.98046875, 1.2470703125, 0.142333984375, -1.759765625, 1.8232421875, -0.58837890625, 0.83984375, -0.16943359375, -0.41015625, 0.281494140625, -0.84326171875, 2.3046875, 2.044921875, 0.59521484375, -0.278076171875, -0.319091796875, 0.87158203125, -0.7275390625, -0.6025390625, -0.445068359375, 1.62109375, -0.41748046875, 1.669921875, -0.84814453125, 1.3095703125, 0.8447265625, -0.67529296875, -1.4814453125, -1.654296875, -0.296630859375, 0.9443359375, 0.27490234375, -1.0068359375, -1.0126953125, -0.93115234375, 1.203125, 0.0122833251953125, -1.48828125, 0.0300140380859375, 0.58349609375, -0.04388427734375, 0.69580078125, -0.360107421875, -0.8603515625, 0.8251953125, 0.9765625, -1.453125, -0.4306640625, 0.8828125, 0.99755859375, -0.1817626953125, 0.037872314453125, 0.248291015625, 0.241455078125, -0.1304931640625, -2.1015625, -0.99462890625, -0.120849609375, 1.361328125, 0.37255859375, -0.2366943359375, 0.256591796875, -0.040985107421875, 0.1771240234375, 0.267822265625, 1.3837890625, -0.9853515625, -0.57666015625, -0.352294921875, -0.99462890625, -0.08575439453125, 0.744140625, 0.93798828125, 0.29248046875, -1.634765625, -0.1644287109375, -0.53759765625, 2.828125, -0.65234375, -0.295654296875, 0.1683349609375, 0.2254638671875, 0.60791015625, -0.20361328125, 0.300048828125, -0.0233306884765625, -0.80712890625, 1.3388671875, 0.94482421875, 0.04925537109375, 1.408203125, -1.2958984375, 0.5390625, 0.053741455078125, -0.004444122314453125, -1.146484375, 1.6689453125, 1.302734375, -0.311279296875, -0.5322265625, 0.491455078125, -0.07183837890625, 1.1201171875, -0.7060546875, 0.89892578125, 0.662109375, 2.9765625, 0.486083984375, 1.0595703125, -0.7109375, -0.97802734375, -1.78515625, 0.393310546875, -0.302978515625, -0.63671875, -1.9375, -0.1572265625, -0.10003662109375, 0.9150390625, 0.9599609375, 0.3203125, -1.169921875, 1.25390625, -0.5078125, -1.4658203125, -0.259521484375, 0.099853515625, -0.437744140625, -1.0107421875, -0.09912109375, -0.83447265625, 2.287109375, 0.061737060546875, -0.5751953125, -0.99560546875, 0.365478515625, 1.2275390625, -1.5703125, -0.80908203125, -0.2412109375, -2.126953125, 1.087890625, 0.70166015625, -0.935546875, -0.78564453125, -1.2734375, -0.41015625, 0.0034027099609375, -0.15478515625, -1.0888671875, 0.2939453125, -0.09039306640625, -0.461669921875, 0.0187225341796875, 0.5927734375, -1.00390625, -0.320068359375, -0.82275390625, 0.6904296875, -1.146484375, 1.013671875, -0.89697265625, -1.365234375, 0.94921875, 0.5361328125, -0.6005859375, 0.24658203125, 0.48779296875, 0.431640625, -0.90283203125, 0.427001953125, 1.802734375, -1.8701171875, 0.154052734375, 1.07421875, -1.1025390625, 0.89404296875, 1.49609375, -0.236083984375, -1.470703125, -0.49658203125, 0.217529296875, 0.7236328125, -1.322265625, 1.9169921875, 0.55029296875, 1.46875, 0.080078125, 0.6728515625, -0.71875, -0.951171875, 0.587890625, -0.14013671875, 0.10516357421875, 0.65576171875, 1.6181640625, 0.15478515625, 0.65576171875, -0.8349609375, 0.166015625, 0.736328125, -0.366455078125, 1.921875, -0.10211181640625, -0.0614013671875, -1.263671875, -1.3935546875, 0.28466796875, -0.038116455078125, -0.055450439453125, 0.256591796875, 0.89599609375, -0.053375244140625, 0.492919921875, 0.243896484375, 0.44677734375, 0.0814208984375, 0.1536865234375, 0.60400390625, 0.7060546875, -1.1416015625, 0.182373046875, -0.428466796875, -0.051116943359375, 0.007366180419921875, 0.1163330078125, 0.06536865234375, 1.1240234375, 1.3037109375, 0.03143310546875, 0.7109375, 0.62939453125, -0.0919189453125, -0.78271484375, -0.058013916015625, 0.3564453125, 0.1695556640625, 1.2197265625, 0.658203125, -1.314453125, -0.1717529296875, 0.1734619140625, -0.4755859375, -1.0244140625, 0.98974609375, -0.447998046875, 1.375, 0.1531982421875, -0.030609130859375, -0.00899505615234375, -0.33984375, -0.048797607421875, -0.12646484375, -0.958984375, -0.00562286376953125, 0.677734375, 0.302001953125, 1.037109375, -0.133056640625, 1.423828125, -0.057342529296875, 0.18701171875, -0.53857421875, -1.0439453125, -2.560546875, 0.57421875, 0.419189453125, -0.939453125, 0.269775390625, -0.1209716796875, 1.2412109375, 1.1337890625, 1.0556640625, 0.2861328125, -1.533203125, -0.900390625, 0.169921875, -0.26611328125, 1.751953125, -0.91455078125, 0.292236328125, 1.4609375, -0.53515625, 1.8701171875, -0.55126953125, 0.794921875, 0.302978515625, 1.3994140625, 0.7744140625, 2.15625, 1.0712890625, 0.276611328125, -1.7236328125, -0.165283203125, 1.935546875, 0.128173828125, -0.2039794921875, -0.4873046875, -0.10888671875, 1.8154296875, -0.3818359375, 2.078125, -0.8662109375, -0.3173828125, 0.53564453125, -1.888671875, -0.984375, -0.5693359375, -0.89697265625, -0.04010009765625, -0.98779296875, -1.2822265625, -0.5693359375, -0.3076171875, 0.9248046875, 1.669921875, -0.205078125, 1.17578125, -0.1920166015625, 2.02734375, 2.525390625, 0.2578125, -0.2313232421875, 0.444091796875, 0.12890625, -0.4853515625, 0.484130859375, 1.607421875, 0.2384033203125, 1.2578125, -0.1634521484375, -0.587890625, 2.05078125, 0.1669921875, -0.12255859375, 0.1787109375, 0.058624267578125, -0.583984375, 1.0849609375, -0.90966796875, 0.414306640625, -0.2493896484375, 0.1705322265625, -1.130859375, -0.91064453125, -0.09326171875, -0.7763671875, 0.070556640625, -0.28857421875, 0.59130859375, 0.09808349609375, 0.296142578125, 0.2457275390625, -0.513671875, -0.67138671875, -0.39892578125, 0.65087890625, -0.266357421875, 0.032989501953125, 1.1279296875, -1.28515625, -0.85546875, -0.798828125, 0.05499267578125, -0.4892578125, -0.51123046875, -1.2314453125, 0.93017578125, -2.369140625, 0.5380859375, -0.62158203125, 0.8330078125, 1.2177734375, 0.31005859375, -0.8232421875, -0.0902099609375, 0.5, 0.0882568359375, 1.12890625, 0.00327301025390625, -0.58544921875, -0.493408203125, -0.00499725341796875, -2.201171875, -0.17919921875, -0.5185546875, -0.48583984375, 0.87841796875, -0.50341796875, 0.26025390625, -1.185546875, 1.4423828125, 0.65673828125, -0.0186614990234375, -1.21484375, -0.55078125, 0.37158203125, -1.234375, -0.826171875, -1.68359375, -0.1341552734375, -0.02813720703125, -0.9345703125, 0.6416015625, 1.1533203125, 1.1591796875, 1.2021484375, -0.294921875, 0.62548828125, 0.94921875, 1.1689453125, -0.29931640625, -0.58447265625, -0.2138671875, -0.059967041015625, -0.6533203125, -0.54150390625, -0.5712890625, 0.018524169921875, 1.7099609375, -1.1806640625, 1.041015625, 0.4404296875, 0.8779296875, -0.96484375, 1.22265625, 0.335205078125, 0.289306640625, 0.61962890625, 0.2255859375, -1.2333984375, -1.9306640625, 2.4140625, -0.646484375, -0.2802734375, 0.82763671875, 0.135498046875, 0.360595703125, 2.259765625, 1.357421875, -0.473388671875, 0.740234375, -0.86083984375, -0.09271240234375, 1.1123046875, -0.52734375, 0.07080078125, -0.98583984375, 1.6201171875, 0.39404296875, 2.140625, -1.6591796875, -1.1845703125, 1.119140625, -1.6103515625, 2.267578125, 0.6005859375, 0.10113525390625, 0.2216796875, -0.7431640625, -0.453369140625, -0.2415771484375, -0.1654052734375, -0.393310546875, -0.44091796875, 0.6806640625, -0.8740234375, 1.31640625, -1.908203125, 0.6396484375, -1.3115234375, 0.5927734375, 0.447509765625, -0.4404296875, -1.55859375, 0.036651611328125, -2.734375, 0.276123046875, 1.484375, -0.18359375, -0.77490234375, -0.81005859375, -0.227783203125, -1.705078125, -0.828125, -0.619140625, 1.521484375, 0.418701171875, -0.11444091796875, -0.9208984375, 0.587890625, 0.76416015625, 0.187255859375, -0.1551513671875, -1.568359375, -1.189453125, -1.154296875, 1.0087890625, 1.3134765625, -0.1475830078125, 0.76953125, -1.09765625, -0.2325439453125, -1.7177734375, -0.195556640625, -1.150390625, -1.19140625, 0.0986328125, 1.4697265625, -0.92236328125, -0.5703125, -0.91357421875, -0.232421875, -0.11871337890625, 1.0712890625, 1.6630859375, -2.51171875, 1.2314453125, 1.013671875, -0.78271484375, -1.0791015625, -0.483642578125, -0.9072265625, 1.5849609375, -0.01093292236328125, 1.3408203125, 0.1212158203125, 0.9189453125, -0.461181640625, 0.568359375, -0.4306640625, 2.451171875, -0.438720703125, -0.892578125, 0.95849609375, -0.363037109375, -1.431640625, 0.352294921875, 1.4873046875, 0.75244140625, -0.310546875, 2.318359375, 2.212890625, -1.01171875, -1.53515625, 1.138671875, -0.7841796875, 0.57080078125, 0.00798797607421875, -0.77685546875, 1.23828125, 1.708984375, -0.1971435546875, -2.3125, -0.58203125, 1.400390625, 0.701171875, 1.1728515625, 0.459228515625, -0.8486328125, 1.0615234375, -1.8681640625, -1.703125, 0.91162109375, -0.52783203125, 0.1497802734375, -0.8037109375, 0.16455078125, -2.177734375, -0.6103515625, -0.29345703125, -1.0986328125, 0.67236328125, -0.0151824951171875, -0.309326171875, 0.47607421875, -0.98291015625, 0.266845703125, -1.7109375, -0.310791015625, -0.79931640625, 0.034942626953125, -1.208984375, 1.9482421875, -0.5888671875, -1.6357421875, 0.7783203125, -1.21875, 0.509765625, 1.2255859375, 0.91650390625, -0.83447265625, 1.1435546875, 0.943359375, 1.4326171875, 0.0177154541015625, 1.4345703125, -0.09405517578125, -1.2578125, -0.396728515625, 1.552734375, 0.0010442733764648438, 0.63818359375, -1.193359375, -0.302734375, 1.03515625, -1.5048828125, 0.6396484375, 0.97607421875, -0.398681640625, -1.5888671875, -0.49755859375, -0.230712890625, -0.09466552734375, -1.6162109375, -0.59814453125, 1.3828125, 1.5693359375, -0.054840087890625, -0.51904296875, -0.5068359375, 0.291015625, 0.30029296875, 1.09375, 2.234375, -0.4443359375, 0.15283203125, -0.11346435546875, 1.1728515625, 0.48486328125, 1.1904296875, 1.4111328125, -0.47021484375, -0.247314453125, 1.48046875, -0.434814453125, 0.96240234375, -1.896484375, -0.1239013671875, -0.755859375, -2.107421875, -1.359375, -0.1973876953125, -1.1845703125, 0.74072265625, 0.3642578125, 0.71875, 0.72900390625, -0.1875, 0.90087890625, 0.62109375, 0.417724609375, 0.2159423828125, 0.2266845703125, 0.1951904296875, 0.986328125, 1.05859375, 1.4248046875, 0.5830078125, 0.55517578125, -0.2237548828125, -1.1103515625, -1.033203125, -0.2646484375, -0.1453857421875, -1.794921875, -1.1484375, 0.806640625, -0.69140625, 2.083984375, -2.115234375, -0.2705078125, 0.57421875, -0.174560546875, 1.6484375, 1.70703125, -0.86279296875, 2.8046875, 0.191650390625, -1.1826171875, 0.43359375, -0.65185546875, -0.95263671875, -0.2305908203125, 0.64111328125, -1.65234375, 0.85888671875, -0.150146484375, -0.0634765625, -0.002933502197265625, -0.02313232421875, -1.4267578125, -0.34765625, -2.564453125, -0.345458984375, 1.0830078125, -0.95654296875, -0.07373046875, -1.46875, 0.74609375, 0.1326904296875, 0.63232421875, 0.501953125, 0.8486328125, -0.1361083984375, 0.8310546875, 0.79345703125, 0.486572265625, 0.99951171875, -1.587890625, -1.494140625, 0.75732421875, 1.6552734375, -1.3623046875, 0.47314453125, 1.0615234375, 1.1044921875, 0.86572265625, -0.195556640625, 0.580078125, 0.9033203125, 0.26904296875, -0.58349609375, -0.2939453125, -1.1279296875, 0.297607421875, -1.7099609375, 0.72119140625, 1.43359375, 1.6279296875, -0.71875, 1.2724609375, -0.5673828125, -0.4912109375, -0.8671875, 1.443359375, 1.6044921875, 1.560546875, -0.225830078125, -0.46044921875, 0.197509765625, -0.00018143653869628906, 1.474609375, -0.141357421875, 0.1982421875, -0.40673828125, -1.1162109375, 1.7333984375, -0.3837890625, -1.111328125, 0.5576171875, -0.71875, -0.03631591796875, 1.5634765625, 1.677734375, -1.064453125, -1.37109375, 0.6357421875, 2.453125, -1.0078125, -0.2861328125, -1.546875, 2.515625, -0.55908203125, 0.75439453125, 0.751953125, 0.31982421875, 1.109375, -0.6171875, 0.1307373046875, 0.1846923828125, -1.5380859375, -2.16015625, -0.30908203125, 0.19677734375, -0.06622314453125, -0.0309600830078125, -1.4130859375, -0.6708984375, 1.7998046875, -1.7216796875, 2.509765625, 0.230224609375, 0.0015249252319335938, -0.442626953125, 0.045318603515625, 0.33447265625, 2.6015625, 0.397705078125, -0.31201171875, 0.51318359375, -0.310791015625, 1.353515625, -0.669921875, -0.44775390625, -0.76220703125, 0.04998779296875, -0.5283203125, -0.12139892578125, 1.4814453125, -0.65283203125, 0.88671875, 1.1328125, 0.80224609375, -0.9736328125, 0.0299835205078125, -0.1553955078125, -0.10003662109375, 0.556640625, -1.490234375, 0.31982421875, -0.005218505859375, -0.0247802734375, 1.8828125, 0.035736083984375, -0.1768798828125, -0.869140625, 0.56689453125, 0.31298828125, 0.54150390625, -0.63720703125, -0.489501953125, -1.1201171875, 0.9150390625, 0.57275390625, -0.90185546875, 0.8330078125, 0.395263671875, 0.055908203125, -0.4912109375, 1.404296875, 0.390625, 1.3876953125, 1.365234375, -0.873046875, -0.82958984375, -0.0972900390625, -1.1025390625, 2.388671875, -0.02142333984375, 0.032958984375, -0.056610107421875, -0.0960693359375, 0.421630859375, 1.439453125, -1.5771484375, -0.100830078125, 0.49609375, -1.45703125, 0.46630859375, 0.0750732421875, -1.4931640625, 1.11328125, 0.6357421875, -0.22607421875, 2.01953125, 0.73388671875, -0.44140625, -0.58837890625, -1.3349609375, 0.35302734375, 0.41064453125, -2.173828125, -0.56591796875, 0.0002925395965576172, 1.30859375, -1.615234375, -0.75146484375, 0.1849365234375, 0.02374267578125, 0.6796875, -0.1337890625, -2.275390625, -1.056640625, 1.0615234375, -0.07122802734375, -0.978515625, 1.2470703125, 0.87353515625, 2.0625, -0.491455078125, 1.818359375, 0.403076171875, 1.349609375, -1.705078125, -1.126953125, 0.9228515625, -0.86669921875, 0.0953369140625, -0.470703125, 1.16796875, -0.255126953125, 0.0180816650390625, -0.4130859375, 1.431640625, -0.79150390625, 1.0068359375, -0.88427734375, -0.30322265625, 0.198486328125, -1.232421875, -0.78076171875, 0.6982421875, -1.5517578125, 0.495361328125, -0.2095947265625, 0.32958984375, 1.369140625, 0.0164031982421875, 0.6103515625, -0.6982421875, -1.70703125, 1.2607421875, 0.10626220703125, 1.1708984375, -0.67724609375, 0.55126953125, 1.802734375, -1.1298828125, 0.3779296875, -0.1370849609375, 0.91015625, 2.171875, -1.146484375, -0.5712890625, 1.7001953125, -0.74951171875, -0.22705078125, -0.332275390625, 0.1109619140625, 1.052734375, -0.8173828125, 1.5615234375, -1.4521484375, 1.412109375, -0.67333984375, 0.9169921875, 1.9326171875, 1.9638671875, 1.193359375, -0.80419921875, -0.316162109375, -0.262939453125, -1.39453125, -0.19677734375, -0.496826171875, 0.168212890625, 1.390625, 0.67138671875, 1.8232421875, 1.3671875, 1.0458984375, 0.068603515625, -0.76708984375, 0.0227508544921875, 0.34228515625, -1.2978515625, 0.9931640625, 0.82470703125, -1.099609375, 0.671875, -1.1748046875, 0.160400390625, -1.4873046875, 0.34228515625, 2.041015625, -0.6064453125, 0.293701171875, 0.55859375, 0.55908203125, -0.1141357421875, -0.48486328125, -1.3193359375, -0.5947265625, -0.0841064453125, 1.728515625, -1.26171875, -0.93896484375, -0.728515625, -0.282470703125, 0.05010986328125, 0.81298828125, -0.85595703125, -0.13623046875, 0.14501953125, 0.0224151611328125, -0.5771484375, -0.552734375, 1.2763671875, -0.72900390625, -0.166748046875, -1.2900390625, 1.4775390625, -0.94873046875, -1.6650390625, -0.3876953125, 0.331298828125, -1.1552734375, -0.442626953125, 0.64453125, 0.0770263671875, -0.125, 1.0859375, -0.9375, 0.89404296875, -0.62841796875, 0.07080078125, -0.98974609375, -0.059539794921875, -1.650390625, 0.53076171875, -0.58740234375, -0.732421875, 1.5712890625, 0.10284423828125, -1.267578125, -1.0146484375, -1.5830078125, 0.348876953125, -1.0185546875, -0.2032470703125, -1.376953125, 0.1671142578125, 1.625, -0.80859375, 2.8515625, 1.0234375, -0.7861328125, 0.9072265625, -0.9521484375, -0.06500244140625, 1.76953125, 0.245849609375, -0.2091064453125, 0.3974609375, -0.415283203125, -0.042205810546875, -0.439208984375, 0.99267578125, 0.58056640625, 1.0615234375, -1.400390625, -1.2451171875, -0.306884765625, -0.10748291015625, 1.96875, 0.00292205810546875, -0.253173828125, 0.33642578125, 0.72900390625, -0.57861328125, 0.60107421875, -0.794921875, -0.8671875, -2.26171875, -0.70361328125, -0.210693359375, -0.77587890625, 1.2509765625, 0.41015625, -0.400634765625, 0.08428955078125, 1.3388671875, 0.389892578125, -0.701171875, -0.30908203125, -0.390869140625, 0.0291900634765625, -0.9599609375, 1.43359375, -1.052734375, -0.75927734375, 0.344970703125, 0.71337890625, -0.465576171875, -0.97412109375, -0.340576171875, 0.99853515625, -1.1328125, 1.63671875, 0.89404296875, 1.3994140625, 1.1552734375, 0.37646484375, -0.424560546875, 1.3896484375, 0.308837890625, 0.049468994140625, 0.381591796875, 0.08001708984375, -0.06854248046875, 0.77197265625, 0.59423828125, 0.10955810546875, 0.395751953125, 0.33837890625, 1.4208984375, -1.4541015625, -0.626953125, 0.5029296875, 0.4833984375, -0.85009765625, 1.1552734375, -0.318359375, 1.470703125, 0.62255859375, -0.164794921875, 1.5078125, -0.0308685302734375, -0.63623046875, 0.0151519775390625, -1.2373046875, -0.50439453125, 1.943359375, -2.548828125, -0.92578125, 1.53515625, 0.01129913330078125, 0.496337890625, -1.095703125, -0.32958984375, -0.45654296875, -0.326416015625, 0.419189453125, 0.286376953125, -0.457275390625, -1.994140625, 0.73828125, 0.61328125, 1.8486328125, -0.638671875, -1.2861328125, -1.091796875, -0.53466796875, 1.4365234375, -0.385986328125, 0.93212890625, -1.8173828125, 0.74365234375, 0.513671875, -0.67724609375, 0.340576171875, 0.376220703125, -0.180908203125, 0.248291015625, -0.150390625, -0.1397705078125, -0.27392578125, 0.0665283203125, 0.0997314453125, 1.0341796875, 0.10198974609375, 0.3779296875, 0.1463623046875, -0.1475830078125, 1.380859375, 0.861328125, -0.0784912109375, 0.0333251953125, -0.1539306640625, 0.281005859375, 1.1591796875, 0.5615234375, -0.4208984375, -0.21826171875, 0.79638671875, -0.7744140625, -0.638671875, 0.85498046875, -0.08563232421875, 2.435546875, 0.8046875, 0.70166015625, -0.3369140625, -0.2103271484375, -0.358154296875, 0.6083984375, -1.466796875, -0.3583984375, 0.26708984375, 0.68505859375, -0.6875, -0.5078125, 1.173828125, -2.1796875, -1.28515625, 1.9384765625, -1.998046875, 1.326171875, -0.33984375, -0.8974609375, -1.7861328125, -1.38671875, -0.5419921875, 0.358642578125, 0.361328125, 0.9033203125, -0.94091796875, -0.92578125, 0.184814453125, 0.4287109375, 0.10272216796875, -0.76611328125, 1.12890625, -1.37890625, -1.349609375, 0.421875, -0.41162109375, 0.7421875, 1.173828125, -0.6083984375, 0.986328125, 0.498291015625, -1.5048828125, -0.041290283203125, -0.564453125, 0.312255859375, 0.0113372802734375, 0.484375, 1.05859375, 1.3310546875, -0.2310791015625, -1.0888671875, 0.95068359375, -0.68408203125, -1.0439453125, 2.25, -0.56787109375, -1.142578125, 0.544921875, 0.7333984375, -1.078125, -0.2110595703125, -0.63427734375, 1.29296875, 1.0400390625, 0.2418212890625, 0.830078125, 0.254150390625, 0.0748291015625, 0.44677734375, -1.076171875, -0.341796875, 0.454345703125, -0.234619140625, -1.5146484375, -0.7509765625, -1.123046875, -0.01290130615234375, 2.546875, -0.76953125, 0.302001953125, 0.08734130859375, 0.90087890625, 1.6689453125, -0.286376953125, -0.9482421875, -0.50439453125, -0.4912109375, 0.974609375, -0.484619140625, -0.11285400390625, 0.7568359375, 0.07470703125, -0.17529296875, -0.1622314453125, 0.338623046875, 1.080078125, -0.30029296875, -1.521484375, 0.2841796875, -0.63623046875, -1.5849609375, -1.9560546875, -0.59521484375, 1.6630859375, -0.27685546875, -1.8818359375, 0.31689453125, -1.33203125, -0.51220703125, -0.109619140625, 0.61083984375, 0.841796875, 0.89208984375, -0.3544921875, 0.117431640625, 0.441162109375, -0.2822265625, 0.6748046875, -0.82666015625, -2.009765625, -1.4296875, -0.07318115234375, 0.81201171875, -0.70654296875, 2.162109375, -0.0438232421875, -0.58935546875, -0.70068359375, 0.425537109375, 0.157470703125, 0.2496337890625, -0.309814453125, 0.650390625, -0.7275390625, 0.0743408203125, -1.26171875, 0.11492919921875, -1.677734375, 0.51513671875, 0.339599609375, 1.0712890625, -0.62109375, 2.177734375, -2.41796875, 0.436279296875, 0.58056640625, 1.072265625, -1.119140625, -1.1513671875, 0.72998046875, -0.004718780517578125, 1.740234375, -1.01171875, -1.4033203125, -0.68212890625, -0.48486328125, 1.611328125, 0.16455078125, -0.95166015625, -0.93798828125, 0.416015625, 1.1171875, 2.328125, 0.391357421875, 0.3857421875, 0.76953125, 1.021484375, -0.4052734375, -1.91015625, -0.2213134765625, -0.493408203125, 0.0614013671875, 0.249755859375, -0.59716796875, 0.165771484375, 0.76513671875, -1.333984375, 0.87353515625, 0.372802734375, 0.026580810546875, -0.75048828125, -0.67041015625, 0.8056640625, -0.02490234375, 1.5185546875, -2.4296875, -2.708984375, 0.317138671875, 0.136962890625, -1.0263671875, -0.00296783447265625, 0.452392578125, 1.0341796875, 0.269775390625, -0.6826171875, 0.285400390625, 1.2236328125, 0.5654296875, -1.6669921875, 0.2457275390625, -0.72509765625, 2.0546875, -1.2470703125, -0.241943359375, 2.42578125, 0.384765625, -0.57958984375, 2.0078125, 0.55517578125, -0.13623046875, 0.8896484375, -0.494873046875, 1.30859375, 0.188232421875, 1.037109375, -1.9326171875, 1.1103515625, -0.135009765625, 0.18505859375, 0.10760498046875, 0.300048828125, 0.9775390625, -0.358154296875, -1.8671875, -0.059539794921875, -0.51513671875, -0.7529296875, -1.0166015625, 1.6171875, -0.415283203125, 0.55859375, -1.0498046875, -0.611328125, 0.045135498046875, -0.810546875, -0.92578125, -0.59716796875, -2.041015625, 0.86669921875, -0.6376953125, -0.31689453125, 0.853515625, -0.050506591796875, -1.2197265625, 0.192626953125, 0.97021484375, -0.368408203125, -0.6171875, -2.15625, -1.443359375, 1.203125, -0.397216796875, -0.44775390625, -0.035186767578125, 1.0546875, 0.7216796875, -0.55029296875, 1.8330078125, -1.4208984375, -0.59130859375, 1.681640625, 0.734375, 0.442626953125, 0.39892578125, 0.5205078125, 0.70166015625, 1.041015625, -2.5625, 0.06109619140625, 1.16796875, -0.390625, 0.24609375, -0.1473388671875, 0.61474609375, -0.185791015625, 1.2373046875, 0.9287109375, 0.061370849609375, -0.59521484375, -0.432861328125, 0.2022705078125, -1.16015625, 0.79833984375, -0.71630859375, -0.218994140625, 0.078125, -2.130859375, 0.2122802734375, 0.342041015625, -0.036285400390625, 0.9990234375, 0.3935546875, 1.4794921875, -0.6103515625, -1.32421875, -0.422119140625, -0.7080078125, 0.046875, 0.346923828125, 0.3486328125, 1.5146484375, 0.82177734375, 0.54736328125, -0.269287109375, 0.323974609375, -1.7529296875, 1.0595703125, -0.81396484375, -0.397705078125, -0.4140625, 0.68505859375, 0.84912109375, -1.2822265625, 1.6357421875, -0.1759033203125, -0.94580078125, 0.4306640625, 0.55908203125, 0.35595703125, -0.1922607421875, 0.447265625, 0.421630859375, 1.1904296875, -0.35595703125, 0.54248046875, -0.6650390625, -1.892578125, 0.65087890625, -0.294189453125, 0.640625, 0.053619384765625, -0.05859375, 1.2373046875, -0.76953125, -0.9755859375, -1.2724609375, 1.548828125, -0.857421875, 1.1337890625, -1.181640625, -1.185546875, 0.477783203125, -0.10650634765625, -1.3583984375, 0.01373291015625, -1.02734375, 0.347900390625, -0.607421875, 1.8173828125, 0.58203125, 1.1787109375, -0.46533203125, 0.1846923828125, 1.4072265625, -0.276123046875, -0.49072265625, 0.95751953125, 0.47900390625, 0.422119140625, 0.03155517578125, 0.78759765625, 0.25732421875, 1.2548828125, -0.322021484375, -0.234130859375, 0.4560546875, 0.31396484375, -0.351318359375, -0.419677734375, -0.56787109375, 1.0302734375, 0.10003662109375, -1.326171875, -1.2529296875, 0.0245513916015625, -0.57666015625, -1.896484375, 0.6171875, 0.52783203125, -0.572265625, -1.791015625, -0.08807373046875, -1.6474609375, -0.22900390625, -1.1005859375, 1.3017578125, 1.873046875, 0.7646484375, -0.07080078125, -0.2249755859375, 1.21875, 0.466064453125, 1.3427734375, -0.7724609375, 0.400634765625, 0.38623046875, 1.5634765625, -1.76171875, 1.2587890625, -0.227783203125, 1.083984375, 0.163330078125, 1.2490234375, 0.2464599609375, 1.0947265625, 0.016265869140625, 0.0013217926025390625, 1.443359375, -0.465087890625, -1.087890625, -2.05078125, -1.2109375, -0.76416015625, -0.18798828125, 0.6640625, 0.40673828125, -0.057586669921875, -0.69775390625, -1.1025390625, -0.029296875, 0.0665283203125, 0.155517578125, 0.29443359375, -0.377197265625, 1.400390625, 1.3544921875, 0.157470703125, 0.42431640625, -1.8173828125, 0.791015625, 1.8583984375, 1.06640625, -0.71728515625, -1.0693359375, -2.71875, 0.12017822265625, 0.31982421875, -0.8193359375, -0.70263671875, 1.2763671875, 2.3203125, 1.0419921875, -0.931640625, -0.2802734375, -0.9853515625, 0.9814453125, 0.330810546875, 0.28125, 0.149169921875, -0.028472900390625, 0.0535888671875, -1.271484375, 0.426513671875, -1.6767578125, 1.17578125, 0.8349609375, -1.2900390625, -1.0263671875, -1.078125, -0.25341796875, 0.935546875, -0.615234375, -0.60986328125, 1.4580078125, 0.01073455810546875, -1.60546875, 0.482666015625, 1.0146484375, -0.92919921875, 0.439453125, -0.94775390625, -0.180419921875, 0.76513671875, -1.56640625, 0.2060546875, -0.54052734375, -0.732421875, -0.405517578125, 0.0194854736328125, -0.326416015625, 0.15380859375, 0.69921875, 0.375244140625, 0.1510009765625, 1.6943359375, 0.61572265625, -0.1190185546875, 0.1934814453125, -0.287109375, -0.884765625, 2.03515625, -0.1614990234375, -0.1483154296875, 0.40771484375, 1.107421875, -0.56494140625, 0.1572265625, -1.740234375, 0.70458984375, -1.1962890625, -0.77099609375, 0.457275390625, 0.11199951171875, -0.447021484375, -1.1669921875, 0.98291015625, -0.666015625, -1.17578125, -0.88330078125, -0.60400390625, -0.85302734375, 1.33203125, -0.1171875, -0.54541015625, -0.8642578125, 0.9072265625, -0.2705078125, -0.76806640625, 1.6416015625, 0.62109375, 0.41162109375, 0.1182861328125, -0.5322265625, 0.1422119140625, 0.2144775390625, 0.307373046875, 0.180908203125, 0.70947265625, -0.6630859375, 0.580078125, 0.5166015625, -0.460693359375, -1.3916015625, 0.7568359375, 0.626953125, 0.254150390625, -0.5908203125, -1.2900390625, -1.1123046875, -1.1748046875, 1.3828125, 0.86279296875, 0.60302734375, 0.285888671875, 1.177734375, 1.8349609375, 0.453857421875, -0.87939453125, -0.736328125, 0.373291015625, 0.82421875, -1.3759765625, -1.97265625, 0.5712890625, -1.13671875, 0.51904296875, 0.623046875, -0.43896484375, -1.216796875, -0.98095703125, 0.218994140625, -0.6640625, 0.98681640625, -1.3271484375, 0.83984375, -1.130859375, -0.65185546875, -0.84521484375, -0.990234375, 1.6005859375, 0.5791015625, 0.2890625, 0.316650390625, 1.380859375, 1.3720703125, -0.274658203125, -0.3212890625, 1.6513671875, -1.2158203125, 0.8681640625, -0.5029296875, 1.376953125, -0.99853515625, -0.6904296875, 0.68408203125, 0.296142578125, 1.2041015625, 0.83154296875, 0.374267578125, -0.56591796875, 1.6494140625, -1.4462890625, 1.3486328125, -0.2471923828125, -1.3017578125, 1.1025390625, 1.4072265625, 0.97998046875, -1.3671875, 0.5595703125, -0.19580078125, -1.515625, 0.44873046875, 1.4697265625, -1.6494140625, 1.544921875, 0.3330078125, -0.35888671875, -0.6640625, 1.0107421875, 0.66650390625, -0.2369384765625, -0.72705078125, -0.0936279296875, 0.485595703125, -1.099609375, -2.380859375, -1.5771484375, -0.17919921875, -0.96826171875, 2.94140625, -0.80224609375, -0.91845703125, -0.4384765625, 0.044403076171875, -0.81201171875, 0.798828125, 1.255859375, 1.99609375, 0.9697265625, -2.41015625, 0.0032939910888671875, -0.55859375, -0.75830078125, 0.3125, 0.445556640625, 0.82470703125, -1.6337890625, 0.053985595703125, -0.07257080078125, -0.2171630859375, 0.499755859375, 1.03125, -0.394287109375, 1.21484375, -2.51171875, 1.4873046875, 1.234375, -0.0118865966796875, -0.140380859375, 1.7744140625, 0.90185546875, -0.921875, -1.197265625, -2.880859375, 1.7314453125, 0.1824951171875, 0.70654296875, 1.9609375, 1.17578125, 0.98681640625, -0.321044921875, -0.68994140625, 1.234375, 1.6201171875, 0.03155517578125, -1.4736328125, 1.12109375, 0.5224609375, -0.94091796875, 0.37744140625, 0.716796875, -1.4609375, 0.254150390625, 0.043975830078125, 1.1953125, -0.340576171875, 0.5, -0.4091796875, -0.71142578125, 0.2626953125, 0.74072265625, 0.82958984375, -0.60791015625, -1.583984375, 0.51513671875, 0.12744140625, -0.95361328125, 1.158203125, 0.005023956298828125, -0.0193328857421875, -1.1455078125, -0.630859375, -2.1796875, 1.05859375, 0.440185546875, 1.02734375, 0.55712890625, -0.58447265625, 0.81298828125, 0.449951171875, 0.92236328125, -0.03753662109375, 0.99755859375, -0.701171875, 0.07379150390625, -0.04290771484375, 1.599609375, 0.07122802734375, -0.1925048828125, -0.329345703125, -2.263671875, 0.1968994140625, -0.26025390625, 0.8916015625, 0.2998046875, -0.332275390625, -0.1253662109375, -1.615234375, 2.15234375, 1.52734375, 2.392578125, 0.1104736328125, 1.2783203125, -0.330322265625, 0.181640625, 0.1641845703125, 1.9169921875, -0.75146484375, -0.175537109375, -0.384765625, -0.06158447265625, 1.15625, -0.404541015625, -0.057220458984375, -0.89453125, -0.21728515625, 0.302978515625, 0.50048828125, -1.36328125, -0.724609375, 0.491455078125, 2.71875, 0.966796875, 0.91357421875, -2.080078125, 1.0224609375, 0.67138671875, 0.485595703125, 1.009765625, -0.70703125, -1.5771484375, 0.78271484375, -0.849609375, 0.98583984375, -0.783203125, -0.66943359375, 1.923828125, 0.1253662109375, 0.58056640625, 0.03466796875, 0.68896484375, 0.171875, -0.6357421875, -1.373046875, -0.88671875, 1.50390625, 1.447265625, 0.441162109375, 0.1375732421875, -0.288818359375, -0.281494140625, -0.7548828125, -0.7802734375, -0.7021484375, 0.91357421875, 1.9970703125, -0.5927734375, 0.48876953125, -1.220703125, 1.0390625, 0.82568359375, -0.4189453125, -0.0877685546875, 0.2236328125, 1.427734375, 0.75341796875, 0.48876953125, -0.10321044921875, 0.211181640625, -0.56103515625, -2.150390625, -0.9306640625, 1.29296875, -0.42529296875, -0.97314453125, -0.1578369140625, -0.055816650390625, 1.5947265625, 1.10546875, -0.37109375, 0.293701171875, -1.033203125, -0.52880859375, -0.2340087890625, -2.416015625, 0.265625, 2.705078125, -1.4375, -0.6669921875, -0.27880859375, 0.865234375, 0.6455078125, 0.0186920166015625, -1.701171875, -1.345703125, -1.6328125, -0.471923828125, 0.623046875, 1.232421875, -0.309814453125, 1.353515625, 0.97119140625, 5.9723854064941406e-05, -1.587890625, -0.08953857421875, -0.410400390625, -0.67333984375, 0.484619140625, -1.38671875, 0.361083984375, -1.1591796875, -0.8271484375, 0.341796875, 0.41259765625, 0.89794921875, -0.2139892578125, -0.053741455078125, -0.880859375, 0.85986328125, -1.02734375, -0.88916015625, 0.1331787109375, 0.184814453125, -1.1552734375, -0.7744140625, 1.1728515625, -0.1185302734375, -0.0391845703125, 1.1240234375, 0.144775390625, 0.84423828125, -0.7783203125, -0.216552734375, 0.9599609375, -0.54638671875, -1.0849609375, 0.552734375, -1.46875, 0.333740234375, 0.1646728515625, 0.68994140625, -1.3603515625, -0.1875, 0.00289154052734375, 0.70361328125, -1.392578125, -0.66796875, 0.30419921875, 0.95166015625, 0.0889892578125, -0.71826171875, -0.23193359375, -0.9033203125, 0.75048828125, -0.490966796875, 1.8466796875, -0.9921875, 0.2247314453125, 0.469970703125, -1.310546875, 0.70947265625, 1.318359375, -1.904296875, 0.42138671875, -0.11431884765625, -0.447509765625, 0.3818359375, 0.0927734375, -0.4384765625, -1.5234375, 0.51708984375, 0.98779296875, -0.45361328125, -1.0625, 0.027252197265625, 0.418212890625, -0.533203125, 0.08770751953125, -1.2470703125, -0.1484375, 0.1773681640625, 0.437744140625, -1.033203125, -0.7734375, 0.404296875, -1.380859375, 0.05987548828125, 0.480712890625, -0.89794921875, -0.98583984375, -0.20166015625, 0.347412109375, 1.35546875, 0.1588134765625, 0.187744140625, -0.03924560546875, -0.57177734375, 0.65625, 0.203369140625, -1.9658203125, -0.377197265625, -1.525390625, -1.05859375, -1.6923828125, -0.1951904296875, -0.62158203125, -1.4580078125, 0.0201568603515625, -0.149658203125, -0.89111328125, 0.006740570068359375, 0.449951171875, 1.326171875, 0.137451171875, 1.2509765625, 0.35400390625, -0.556640625, 0.310546875, 0.72216796875, -0.171875, 0.0648193359375, 0.94482421875, 0.04034423828125, 0.53515625, 1.0556640625, -0.31689453125, 0.282470703125, -0.7412109375, 0.369384765625, -0.438720703125, 0.335693359375, 0.5751953125, 1.0830078125, -0.31103515625, -1.0107421875, 0.38916015625, 0.51171875, -1.3447265625, -0.302490234375, 0.697265625, -0.1943359375, -1.123046875, -1.8349609375, 0.11077880859375, 0.4755859375, 0.056610107421875, 0.28271484375, 0.2430419921875, 1.4423828125, 0.79345703125, 3.166015625, -0.28515625, 1.08984375, 0.490478515625, 1.021484375, -0.5341796875, 0.14599609375, 0.338134765625, -1.0888671875, -0.402587890625, -1.0859375, 0.1903076171875, 0.228759765625, 0.12359619140625, 2.162109375, 0.62548828125, -0.923828125, -1.5947265625, -0.065673828125, -0.287841796875, 1.6142578125, -0.02569580078125, 0.269287109375, 0.97705078125, -0.66259765625, 1.2744140625, -0.77392578125, 0.296142578125, 1.2431640625, -1.189453125, 1.177734375, -0.61962890625, -1.5224609375, 0.1962890625, -1.55078125, 0.060272216796875, 0.76806640625, -1.2412109375, -1.033203125, -0.96240234375, -0.85546875, -0.82568359375, -1.1025390625, 0.251953125, -0.07183837890625, 2.345703125, 0.27880859375, -0.9365234375, 0.41552734375, 0.79638671875, -0.2578125, 0.6630859375, 0.56005859375, -0.57861328125, -0.56982421875, 1.8955078125, 0.486328125, 0.64794921875, -1.2841796875, -0.4892578125, 0.96630859375, 1.22265625, -0.097412109375, -0.367431640625, 0.412109375, 0.43798828125, -0.84619140625, 1.287109375, 0.89599609375, -0.82177734375, 0.76171875, 0.06048583984375, 0.646484375, 1.4033203125, -0.2327880859375, 0.37841796875, -0.432373046875, 0.1217041015625, 0.475830078125, -0.0517578125, -0.10009765625, -0.814453125, 0.53173828125, -1.564453125, -1.2353515625, -0.5048828125, -0.7138671875, -1.3046875, -1.09375, -0.54052734375, 0.8955078125, -0.95166015625, 0.2109375, 1.166015625, 0.326904296875, -0.2890625, 0.66650390625, -0.309814453125, -0.04351806640625, 1.3046875, -0.56884765625, 0.93798828125, 0.379638671875, -1.8056640625, -1.001953125, 0.791015625, -0.55615234375, 1.7607421875, 0.76123046875, 0.353759765625, 1.5634765625, 1.1650390625, 0.67236328125, -0.2900390625, 1.6015625, -0.9306640625, -1.255859375, 0.86083984375, 0.12841796875, 0.1702880859375, -0.45068359375, 1.6875, -0.411865234375, -0.802734375, 1.75390625, 2.462890625, 1.009765625, 0.703125, -0.35400390625, -0.74365234375, -2.27734375, -0.2432861328125, 0.77490234375, -1.9375, -1.6650390625, 1.037109375, 0.266357421875, -0.5048828125, 0.0156402587890625, 0.432373046875, 0.2783203125, 0.39599609375, 0.166259765625, 0.79541015625, 0.837890625, 0.334228515625, 0.448974609375, -0.76611328125, -1.779296875, -0.00447845458984375, -0.83251953125, -0.74365234375, 0.07781982421875, 0.96435546875, -0.4833984375, -1.7783203125, 0.26953125, -1.21875, -0.80419921875, 1.2451171875, 0.5244140625, 0.28271484375, 0.7802734375, 0.76416015625, -1.361328125, 0.3271484375, 1.5517578125, -0.85595703125, -0.515625, -1.0615234375, -0.8818359375, -0.75830078125, 1.4375, 0.449951171875, 0.6083984375, 1.1337890625, -0.241455078125, 1.71484375, -0.131103515625, 0.90673828125, 0.95263671875, 1.7470703125, -1.0751953125, 0.25146484375, 0.6943359375, -0.10919189453125, 0.33251953125, 0.37890625, -0.84228515625, 0.284912109375, -0.271484375, -0.865234375, -1.8583984375, -0.7099609375, 0.11865234375, 0.484619140625, 0.1802978515625, -1.673828125, -0.195068359375, -0.87548828125, -0.416015625, -0.88671875, -0.763671875, 0.1475830078125, 0.118896484375, 0.100830078125, -0.433349609375, -0.2213134765625, 0.444580078125, 0.37939453125, 1.1533203125, -0.254150390625, -0.1807861328125, 0.99658203125, -0.2080078125, 0.45263671875, -0.1839599609375, 0.2020263671875, 0.890625, -1.6181640625, -1.134765625, 0.9951171875, 1.609375, -0.236572265625, 0.323486328125, -0.43798828125, -0.380126953125, 0.0888671875, -0.86376953125, 0.321044921875, -1.0517578125, 0.505859375, -0.8232421875, -0.173583984375, 0.5908203125, -0.144287109375, -0.342041015625, -0.970703125, -2.130859375, -0.7802734375, 1.34375, -0.1480712890625, 0.0228729248046875, 1.1259765625, 0.12176513671875, -0.87451171875, -0.322509765625, 0.63916015625, 0.06304931640625, 0.5615234375, -0.654296875, 0.300537109375, 0.1429443359375, -0.429931640625, -0.1871337890625, -1.736328125, -0.59716796875, -0.16650390625, 0.175048828125, 0.8427734375, 0.89892578125, -0.16015625, 0.464111328125, 0.6806640625, -0.82666015625, 0.101806640625, 0.29150390625, -0.44140625, -0.085693359375, -0.8544921875, -0.87646484375, -0.96630859375, 1.54296875, -0.71923828125, 0.06524658203125, -0.6357421875, -0.19189453125, 1.322265625, -1.2646484375, -0.189208984375, -0.36376953125, 0.9892578125, 0.75048828125, 1.3955078125, -0.01143646240234375, -0.30126953125, 2.138671875, -0.92529296875, -0.1585693359375, -0.08233642578125, 0.137451171875, -0.359130859375, -2.2578125, -1.8544921875, 0.744140625, 0.422119140625, 1.5263671875, 0.4248046875, -0.7978515625, 0.31689453125, -0.2115478515625, -2.08203125, -0.57568359375, 0.671875, -1.736328125, -0.751953125, 0.2822265625, 1.7431640625, 0.2406005859375, -0.205322265625, -0.58154296875, 0.595703125, 0.70068359375, 0.92626953125, -0.79248046875, 0.273193359375, -0.06365966796875, -0.11993408203125, -0.63232421875, -1.7392578125, 0.02447509765625, 2.767578125, -0.66015625, 0.78857421875, -0.08184814453125, 0.2288818359375, -0.038238525390625, 0.490966796875, -0.2210693359375, -2.818359375, -1.634765625, 0.465087890625, 0.68017578125, 1.5361328125, 1.255859375, 0.69384765625, 0.1715087890625, -0.83837890625, 1.1943359375, -1.9482421875, 0.80810546875, -0.4150390625, 0.474609375, -1.8564453125, -0.131103515625, 0.8623046875, -0.75537109375, -0.29443359375, 0.341796875, 0.4306640625, -1.2578125, 0.3359375, 1.0048828125, -0.315673828125, -0.1788330078125, -1.04296875, -1.6455078125, 0.56494140625, -0.349365234375, 0.06365966796875, 0.1553955078125, -0.544921875, 1.7578125, -0.892578125, -0.093994140625, -1.099609375, 1.037109375, -1.9208984375, -0.6904296875, -0.1549072265625, -0.156494140625, 1.0, 0.52587890625, -0.006343841552734375, 0.57470703125, 1.5166015625, 0.8349609375, -2.005859375, -0.5166015625, -0.53076171875, -1.1171875, 0.81494140625, 1.8779296875, 0.205078125, 0.73779296875, -0.4150390625, -0.7646484375, -0.280029296875, 1.6025390625, 1.7021484375, 1.630859375, -0.849609375, -3.13671875, 0.5908203125, -0.37548828125, 1.8798828125, 0.677734375, 1.150390625, 0.393798828125, -0.79248046875, -0.2646484375, -1.4658203125, -1.353515625, -1.1435546875, 0.1624755859375, -0.1966552734375, 0.771484375, 0.89990234375, -0.0657958984375, -1.0810546875, 0.6484375, -0.433837890625, -0.08050537109375, 0.1507568359375, -0.164306640625, 1.0341796875, -0.57421875, 0.302001953125, -0.147216796875, -1.9736328125, 0.505859375, 0.1878662109375, 0.10107421875, -0.41064453125, -0.265869140625, -0.39453125, -1.666015625, 1.27734375, -0.243408203125, -0.58984375, 0.974609375, 0.62451171875, 0.2900390625, -0.39990234375, -1.615234375, -1.2392578125, 0.55517578125, -2.1953125, 0.291748046875, -1.1796875, -1.1796875, 1.509765625, 0.09417724609375, 0.198486328125, 2.03125, -1.345703125, 1.65625, -0.662109375, 0.9765625, 1.1513671875, -0.734375, -0.05072021484375, -1.673828125, -0.58984375, -0.020111083984375, 0.8779296875, -0.775390625, -0.326904296875, -0.6298828125, 0.98681640625, -1.1044921875, 0.5224609375, -0.2353515625, 0.003467559814453125, 1.001953125, 0.6962890625, 0.63671875, -0.529296875, 1.4267578125, -0.564453125, 0.44287109375, -0.984375, 1.3076171875, 1.240234375, 0.92236328125, 0.90771484375, -0.060455322265625, 0.66650390625, -0.99072265625, -1.056640625, -0.7666015625, -0.72705078125, -0.81640625, 0.92431640625, -2.8828125, 1.1640625, -0.0997314453125, -1.087890625, -1.8837890625, 1.931640625, -1.3056640625, -0.46142578125, 0.03802490234375, 0.8154296875, -0.2457275390625, -0.2393798828125, -1.404296875, -1.333984375, -0.45458984375, 0.5400390625, -0.57080078125, 1.33203125, 1.296875, 0.238037109375, 0.857421875, 0.280517578125, 0.86376953125, -1.2919921875, 0.2154541015625, -0.1405029296875, -0.368896484375, -1.6240234375, -0.583984375, 0.21240234375, -0.52783203125, -0.78369140625, -2.71875, 1.5205078125, -0.72021484375, -1.2041015625, -0.105224609375, -0.52587890625, 1.2314453125, 0.194580078125, 2.087890625, 1.30078125, 1.0673828125, -0.07965087890625, -0.397216796875, -1.0341796875, 0.2271728515625, -0.88330078125, -0.81884765625, 0.042327880859375, 0.1842041015625, -0.79541015625, 0.6220703125, -0.460205078125, -0.1527099609375, -1.6689453125, -0.05889892578125, -0.4345703125, -0.196044921875, 0.0948486328125, 0.55908203125, 0.45947265625, -0.0264129638671875, -1.1083984375, 0.37255859375, 1.353515625, 0.306884765625, -0.66943359375, -1.169921875, 0.373046875, -0.047882080078125, 1.5546875, 0.319091796875, -1.7646484375, 0.5869140625, -1.150390625, 1.12109375, 0.36279296875, 0.03790283203125, -1.1611328125, 0.351806640625, 2.267578125, -0.7841796875, 0.126220703125, 1.3564453125, -0.1844482421875, -1.322265625, -0.89453125, -0.7060546875, 0.415771484375, 2.4140625, -0.861328125, 0.93603515625, 0.35009765625, -1.0625, -0.294677734375, 0.5576171875, -1.6044921875, -0.65087890625, -0.77978515625, 0.349609375, 0.10931396484375, 0.9541015625, 0.203857421875, 0.107177734375, -0.344482421875, 1.3798828125, 1.0068359375, 1.2041015625, -1.2001953125, 0.95751953125, 1.525390625, 2.103515625, -0.11328125, 0.6171875, 0.7685546875, -0.430908203125, -0.6728515625, 0.298583984375, -0.375, 0.58740234375, -0.77294921875, 1.341796875, 0.1910400390625, -0.5224609375, 0.25244140625, -0.9619140625, 0.27587890625, -2.19140625, -0.429931640625, -0.48046875, 2.0859375, 0.62158203125, 1.005859375, 0.1495361328125, -0.04071044921875, 0.53125, -1.3369140625, 0.047637939453125, -0.28076171875, 0.3955078125, 1.0712890625, 0.161376953125, 0.84619140625, -0.09552001953125, 0.75732421875, -0.74072265625, 1.1064453125, 0.479248046875, 0.11297607421875, -1.2294921875, 1.7490234375, -1.98828125, -0.384521484375, -0.74951171875, 2.01171875, -0.08599853515625, -0.22412109375, -1.1376953125, -0.57421875, -0.671875, -0.430419921875, 0.28271484375, 1.6630859375, 0.8349609375, -0.48828125, -0.50732421875, -1.07421875, -1.4052734375, 0.46240234375, 0.58154296875, -1.103515625, -0.319580078125, -0.58251953125, -0.01303863525390625, -1.591796875, 1.62109375, 0.139404296875, -0.448974609375, -0.1622314453125, -0.8828125, -0.677734375, -0.53125, -0.08795166015625, 1.2451171875, 0.13720703125, -0.5322265625, 0.35693359375, -0.779296875, 1.1416015625, 1.5380859375, -2.203125, -0.44140625, -0.6826171875, -0.30712890625, -0.34765625, -0.097900390625, -0.30224609375, -0.6298828125, -0.2705078125, -0.427734375, 0.48876953125, -1.2294921875, -0.298095703125, -1.150390625, -0.65478515625, 0.69873046875, -0.818359375, 0.7880859375, -0.98095703125, 0.53173828125, 0.42724609375, 0.36962890625, 0.8671875, 0.365478515625, 0.042236328125, 2.556640625, -0.6630859375, 1.0791015625, 0.1273193359375, -1.357421875, -1.5478515625, -2.33984375, 0.369873046875, 1.2109375, -0.35498046875, 0.66796875, -1.26953125, 0.3525390625, 0.34814453125, 0.6435546875, 0.44140625, -1.9091796875, 0.299560546875, -0.39404296875, -1.689453125, -2.021484375, -1.3349609375, -1.7470703125, 0.466064453125, 1.056640625, 0.308349609375, 0.397216796875, -0.1455078125, -0.326171875, -1.50390625, -0.8837890625, -1.060546875, 1.00390625, -1.6640625, 1.2314453125, 1.0234375, -1.58984375, -0.233154296875, -0.281494140625, 0.165771484375, 1.3037109375, 0.490234375, -1.6494140625, 0.6005859375, -0.11810302734375, -0.391845703125, -0.2103271484375, -0.043182373046875, 0.8349609375, -0.79296875, 2.271484375, 2.13671875, -0.0263824462890625, 0.47265625, 1.328125, 0.494140625, 0.626953125, -0.341064453125, -0.2548828125, -0.77294921875, -0.916015625, -0.70703125, 1.125, -0.212158203125, 1.6025390625, -1.5322265625, 0.3173828125, -1.4755859375, -0.83447265625, 0.52490234375, -1.35546875, -0.6806640625, -0.38232421875, 1.6640625, 1.9853515625, 1.2392578125, -1.0927734375, 0.49755859375, -0.5205078125, 1.3310546875, 1.0908203125, 0.931640625, 0.09539794921875, -0.61328125, 0.150634765625, -0.541015625, -2.681640625, -1.0791015625, 0.82421875, 0.180419921875, 2.19921875, 1.021484375, 0.75390625, -0.45703125, -0.615234375, -0.68505859375, -0.341552734375, -0.52685546875, -0.2354736328125, -0.54638671875, -1.5205078125, 0.7353515625, -0.82568359375, 1.4736328125, -0.419677734375, 0.1932373046875, -0.724609375, 2.560546875, -0.06201171875, 1.2666015625, 1.7822265625, -0.88916015625, 0.00833892822265625, 0.6435546875, 1.3974609375, 0.51904296875, -1.75390625, 0.416259765625, 0.29736328125, 0.74560546875, -0.6611328125, -1.1982421875, -0.50341796875, 0.7333984375, -0.77099609375, 2.263671875, 1.162109375, -1.5029296875, -0.016510009765625, -0.367919921875, 0.1541748046875, 0.060028076171875, 1.11328125, 1.0341796875, 0.323974609375, -0.97509765625, 0.86669921875, -1.81640625, -0.11456298828125, -0.8466796875, -0.75537109375, -0.880859375, 0.0716552734375, -0.31005859375, -0.62060546875, 0.84033203125, 0.87451171875, -0.57373046875, -1.0556640625, 1.0263671875, -0.35693359375, 0.0408935546875, -1.189453125, 0.0036487579345703125, 0.83642578125, -0.0171356201171875, -0.8642578125, -1.21875, -0.52880859375, -0.0947265625, 1.2470703125, -1.0361328125, -2.232421875, 1.0595703125, -0.7783203125, -0.810546875, -0.281494140625, -0.7041015625, -0.219970703125, -0.92041015625, -0.46728515625, -0.30126953125, 0.43017578125, 0.1646728515625, -1.0146484375, -1.962890625, 0.515625, -2.876953125, -1.5478515625, 0.128662109375, -0.88232421875, 1.2412109375, 0.350830078125, -0.884765625, 0.7333984375, -0.03857421875, -1.982421875, -0.52880859375, -0.3896484375, 0.1768798828125, -0.09649658203125, 1.0927734375, -0.6064453125, -0.29736328125, -0.135986328125, 1.3603515625, 0.11773681640625, 1.498046875, -0.5732421875, -1.6494140625, 0.423095703125, 1.1484375, 1.330078125, 1.4150390625, 0.337890625, -0.84765625, 0.77783203125, 1.0498046875, 0.385009765625, -0.31396484375, 1.1875, -0.2103271484375, 0.1988525390625, 2.16015625, -0.7890625, 0.09844970703125, 1.083984375, 0.44921875, -1.66015625, -1.27734375, 0.2861328125, -0.313720703125, -1.48046875, 0.1729736328125, 0.927734375, 0.363525390625, -0.2471923828125, -0.336181640625, 0.2763671875, 0.703125, 1.763671875, 0.30859375, -1.1376953125, -0.0251312255859375, -0.461181640625, 1.6826171875, -0.79638671875, -1.072265625, 0.671875, -1.2138671875, 0.036773681640625, -0.814453125, -0.297119140625, 0.1806640625, -0.10302734375, 1.0869140625, -0.26416015625, 0.19287109375, 0.083740234375, -0.372314453125, -0.07147216796875, -0.69580078125, 0.5107421875, 1.76953125, 0.479736328125, -1.791015625, 0.94775390625, 0.6044921875, -1.3984375, -1.513671875, -1.201171875, -2.0625, 1.1767578125, -0.6181640625, -0.491455078125, 0.254638671875, 0.51513671875, -0.324951171875, -0.050506591796875, 0.1759033203125, -0.46923828125, -0.93994140625, -0.90869140625, 0.12164306640625, 2.55859375, 0.455078125, -0.82470703125, 1.8759765625, 1.6884765625, -0.62548828125, -0.205078125, -2.099609375, 0.1693115234375, -0.06475830078125, 0.7060546875, -0.32421875, -0.8818359375, 1.5107421875, -1.0869140625, -0.77685546875, 1.998046875, -0.658203125, -0.5615234375, -0.60302734375, -0.533203125, -0.0650634765625, 0.09857177734375, -0.97802734375, 0.210693359375, 0.3466796875, -1.876953125, 0.34619140625, -1.228515625, 0.533203125, 0.182861328125, 1.345703125, -1.724609375, 0.444091796875, 1.095703125, -0.50439453125, -0.049285888671875, -0.68115234375, -0.06689453125, 0.381103515625, -2.361328125, -1.0029296875, -1.2958984375, 0.89404296875, -0.71533203125, -0.0980224609375, -0.56396484375, 0.98779296875, -2.015625, 0.578125, 0.2384033203125, -1.365234375, 1.6611328125, 0.07025146484375, 0.452880859375, 0.352294921875, -1.3681640625, -1.4375, -0.0077972412109375, 0.28955078125, 1.1865234375, 0.40185546875, -0.359130859375, -0.34375, -0.61767578125, -0.0682373046875, 1.216796875, -0.7900390625, -0.01812744140625, -0.78173828125, -0.9033203125, 1.419921875, -0.157470703125, 0.77294921875, 2.115234375, 0.095703125, -0.7568359375, 0.54638671875, 0.42236328125, 0.9541015625, 0.414794921875, 0.375244140625, 1.3701171875, -1.0234375, 0.98876953125, 0.390380859375, 0.8642578125, -1.216796875, -0.923828125, -0.37548828125, -0.234375, -1.5703125, 0.3828125, 1.02734375, 0.2003173828125, 0.376953125, -1.1240234375, -0.619140625, -0.52294921875, -1.3212890625, -1.345703125, -0.525390625, -1.2333984375, -1.2734375, -0.14208984375, -0.47998046875, -0.7998046875, -0.43798828125, 1.177734375, -1.0078125, 1.4365234375, 1.640625, 1.7099609375, 0.72216796875, 0.3466796875, 0.66650390625, -0.56201171875, -0.71630859375, 0.3515625, -0.5966796875, -0.1820068359375, -1.8173828125, -1.349609375, -1.3642578125, 1.8232421875, 1.75, 1.009765625, 1.416015625, 0.1861572265625, -0.47119140625, -0.87939453125, -0.626953125, 0.484375, -1.5732421875, 0.63427734375, -1.3984375, -1.3818359375, 0.08514404296875, 0.191650390625, 0.403564453125, 0.186279296875, -1.87890625, -0.94091796875, -1.12109375, -1.203125, 0.046112060546875, 0.69970703125, -0.5927734375, -0.12188720703125, -1.078125, -0.85791015625, -0.060943603515625, -0.48828125, -0.01493072509765625, -0.59375, 0.685546875, -1.10546875, -0.955078125, 0.4833984375, 1.3125, -0.859375, 0.338134765625, 1.31640625, 0.37353515625, -0.428955078125, -0.97998046875, 1.375, -0.1318359375, -1.4033203125, -0.03558349609375, -1.1572265625, -1.5478515625, -0.445556640625, 1.302734375, -0.86474609375, -0.423828125, -0.77099609375, 0.33642578125, 0.720703125, -0.402587890625, -0.32421875, -0.1229248046875, 1.5322265625, -0.1058349609375, -0.48193359375, 0.84912109375, 1.4140625, -0.5302734375, 0.716796875, 0.3330078125, 0.5576171875, 0.5830078125, 0.70849609375, 1.287109375, 0.300048828125, -0.497314453125, 0.29736328125, 0.01284027099609375, 0.54052734375, 0.61865234375, -0.0888671875, 0.419677734375, 0.67138671875, -0.94921875, -0.8037109375, -0.53271484375, 0.0228729248046875, 0.73779296875, -0.96484375, 0.2484130859375, -1.5966796875, 0.389404296875, -2.1171875, 0.8779296875, -0.66748046875, 0.385986328125, 1.44140625, -2.248046875, -2.09375, 0.62841796875, 0.5771484375, 0.5947265625, 0.463623046875, 1.513671875, -0.66015625, -4.15625, 0.57470703125, 0.2061767578125, 0.84521484375, 1.8759765625, 0.2431640625, -1.9111328125, -1.41796875, -0.33984375, -0.142333984375, -0.369384765625, -1.4306640625, -0.5634765625, -0.133056640625, -1.033203125, 0.6279296875, 1.0908203125, -0.623046875, 0.859375, -0.054351806640625, 1.5, 0.63037109375, -0.4052734375, -0.50439453125, -0.1689453125, 0.282470703125, -0.65869140625, 1.4638671875, -0.16650390625, 0.650390625, 0.476318359375, -0.52587890625, 0.73779296875, -0.89697265625, -1.5390625, 0.408203125, 0.1978759765625, 0.338623046875, 0.443359375, 1.2802734375, -0.403076171875, -0.21630859375, 1.4384765625, -1.1513671875, -0.939453125, 0.75341796875, 0.177734375, -0.69091796875, 1.1044921875, -1.044921875, -0.37353515625, -0.09814453125, 0.382568359375, -0.509765625, -0.0791015625, -1.2861328125, -0.1549072265625, 1.2666015625, -0.7412109375, 1.2470703125, 0.2191162109375, 1.4951171875, 0.2783203125, 0.54052734375, 2.037109375, -1.09765625, -0.239990234375, -0.0919189453125, 0.26611328125, 1.0283203125, -0.3154296875, 0.60546875, 0.00856781005859375, 0.00201416015625, -1.482421875, 1.4453125, 0.171630859375, 1.701171875, -1.2548828125, 0.80517578125, -1.265625, -0.419677734375, -1.787109375, -0.422607421875, 0.17138671875, -0.84619140625, 0.2293701171875, 1.005859375, 0.72265625, 0.1798095703125, -0.79150390625, 0.552734375, 0.916015625, 0.97900390625, 0.76904296875, 0.6884765625, -1.9638671875, 1.28515625, 0.68798828125, -0.2322998046875, 0.849609375, 1.240234375, -0.50732421875, 0.66357421875, -1.0927734375, 1.8486328125, 0.241943359375, -0.5927734375, 0.77734375, 0.91650390625, -1.787109375, 1.763671875, 0.7470703125, -1.3330078125, 0.2144775390625, -0.10736083984375, 0.5146484375, 0.91064453125, -0.399169921875, -0.342529296875, 0.1966552734375, -0.7509765625, -2.142578125, 2.501953125, 0.0787353515625, 0.264892578125, 1.0419921875, 0.2447509765625, 0.25537109375, 1.017578125, -0.82470703125, 0.374755859375, -0.6982421875, 0.27734375, -0.436767578125, 0.76025390625, -0.32177734375, -0.88427734375, -0.69189453125, 0.94775390625, -0.4296875, -1.0849609375, 0.23193359375, 0.3359375, 1.44140625, -0.56494140625, 0.53076171875, 0.51416015625, 0.395263671875, -1.359375, 0.12335205078125, 0.7080078125, -0.552734375, -0.427978515625, -1.474609375, -0.226318359375, 0.01415252685546875, 0.54443359375, -0.533203125, -0.40625, 2.310546875, -0.136962890625, 1.6005859375, 1.3369140625, 0.0009222030639648438, 0.40625, 0.338134765625, -0.40478515625, -0.1866455078125, 0.039825439453125, 3.12890625, 0.6640625, -0.356689453125, 0.826171875, -0.958984375, -0.10009765625, 1.255859375, 0.07049560546875, 1.046875, 1.1630859375, 2.83984375, -0.9638671875, 0.8779296875, -0.52880859375, 1.5703125, -1.2490234375, 0.039642333984375, -0.07720947265625, 1.126953125, 0.3740234375, -1.037109375, 1.625, -0.849609375, 1.30859375, 0.85986328125, 0.1424560546875, -0.30712890625, -0.400146484375, -0.33349609375, -1.697265625, 0.321533203125, 0.7939453125, -1.1201171875, 1.080078125, -1.1943359375, -0.09442138671875, 1.30078125, 0.55078125, 0.18701171875, -1.2900390625, 0.0195159912109375, 0.966796875, -0.254638671875, -0.95068359375, -0.35791015625, 0.31591796875, 0.66259765625, 0.32373046875, -1.5712890625, -0.1375732421875, 0.239501953125, -0.1893310546875, -0.74658203125, -0.2939453125, -0.7265625, -0.64990234375, 0.82470703125, -0.2462158203125, 0.257568359375, 0.9345703125, -0.0081024169921875, 0.09112548828125, 0.81591796875, -0.6005859375, 1.3134765625, 0.2802734375, 0.5166015625, -0.8291015625, -0.8232421875, 1.03125, -0.51416015625, 0.31298828125, -0.08892822265625, -2.857421875, -0.0718994140625, 1.080078125, -1.5458984375, -0.0308380126953125, -1.04296875, 1.013671875, -0.6904296875, -0.1416015625, 1.751953125, 0.31396484375, 1.4619140625, 1.54296875, -0.273681640625, -0.67236328125, -0.2186279296875, 1.1982421875, -0.59619140625, 0.81689453125, 0.231689453125, -0.347900390625, 0.21142578125, 0.1337890625, 1.5615234375, 0.89990234375, 1.3671875, -0.334716796875, -0.58447265625, 1.2158203125, 1.0673828125, -1.859375, 0.360595703125, -1.1962890625, -0.375732421875, -0.31689453125, 1.3095703125, -0.18603515625, -0.047393798828125, 1.3681640625, 0.352783203125, -0.16650390625, 1.240234375, -0.3125, -1.5205078125, -0.10479736328125, 0.77783203125, -1.15234375, -1.2109375, 0.20166015625, 0.33935546875, 0.7421875, -0.53369140625, 0.178955078125, 0.2149658203125, 1.857421875, -1.0986328125, -0.250244140625, 0.51416015625, 0.474853515625, 0.97998046875, 1.599609375, -0.39794921875, -1.9287109375, 1.4501953125, -0.609375, 1.5400390625, 0.5517578125, 0.96728515625, -0.603515625, 0.03948974609375, -0.277587890625, 0.0164947509765625, 2.0234375, -0.88134765625, 0.047698974609375, 0.271484375, -0.35595703125, 1.271484375, 0.7626953125, -0.6826171875, 0.07122802734375, 0.307373046875, -1.166015625, -1.3017578125, -0.4501953125, -1.15625, 0.404296875, 1.1875, 1.1318359375, 0.451171875, -0.185791015625, -0.57861328125, 0.9765625, 0.58056640625, 0.83984375, -1.044921875, 0.401123046875, 1.380859375, -1.5380859375, -0.75732421875, 1.01171875, 0.51171875, -0.64306640625, 0.9111328125, -0.2193603515625, -0.9833984375, 0.190673828125, -0.642578125, 1.3095703125, -0.6904296875, 0.09783935546875, 0.29345703125, 0.5966796875, 0.1396484375, -0.80078125, -0.951171875, 1.439453125, -0.40087890625, 0.12481689453125, -0.01042938232421875, -1.908203125, 0.396484375, -0.29052734375, 0.06488037109375, 0.42724609375, -0.70947265625, -0.8466796875, -0.19384765625, 0.25537109375, -0.473876953125, 2.802734375, -0.462158203125, -1.455078125, -0.433349609375, 1.01953125, -0.8935546875, 0.023529052734375, -0.521484375, -0.468505859375, 1.3115234375, 1.333984375, 0.705078125, -0.397705078125, 1.1142578125, 1.69140625, 0.7080078125, 0.6123046875, -0.35302734375, -0.65673828125, -0.55078125, 1.0703125, -0.2288818359375, 1.1640625, -0.810546875, -1.384765625, -0.7607421875, -0.29736328125, -1.451171875, -1.1708984375, 1.4833984375, -0.357421875, -0.85009765625, 0.853515625, -1.6181640625, 0.1116943359375, -1.671875, -1.55078125, -1.0576171875, -0.78759765625, -2.203125, -0.406982421875, -1.2177734375, 0.11932373046875, -0.178466796875, -1.423828125, -1.4248046875, 1.173828125, 0.5458984375, -0.9814453125, -0.09576416015625, -1.482421875, 0.424560546875, -0.89453125, -1.0546875, -0.7900390625, 1.1533203125, 1.8623046875, 0.9873046875, -0.57421875, -0.2303466796875, -0.72265625, -0.274169921875, -0.662109375, -0.912109375, -1.248046875, -0.1654052734375, -0.8984375, 0.351318359375, -0.245361328125, 0.50244140625, 1.119140625, 0.1187744140625, 0.0880126953125, -1.3525390625, -0.857421875, 0.52587890625, 1.537109375, 0.2685546875, 1.6904296875, 0.10888671875, 0.85302734375, -1.6884765625, -0.4443359375, -0.79931640625, -1.220703125, -0.353759765625, -0.537109375, -0.74755859375, -0.0885009765625, 0.1771240234375, 0.1009521484375, -0.58740234375, 0.377197265625, -0.0750732421875, 0.84375, -0.72705078125, 1.2080078125, 0.60107421875, 0.035369873046875, -1.154296875, -0.998046875, -1.4677734375, -0.7880859375, 0.8662109375, 2.0, 0.82958984375, -0.06103515625, 1.1494140625, -0.5595703125, -1.9873046875, -1.2158203125, 0.75146484375, -1.1669921875, 0.79248046875, 0.95068359375, -0.19970703125, -2.068359375, 1.5234375, 0.175048828125, -0.246826171875, -1.6982421875, -1.1962890625, -1.1572265625, 1.6171875, -2.068359375, 0.89794921875, 0.94580078125, -0.039306640625, -0.1522216796875, 1.3369140625, -1.3740234375, -1.58203125, 0.64990234375, -0.1009521484375, 1.783203125, -0.9130859375, 0.697265625, 1.033203125, 1.0439453125, 0.07794189453125, -0.56982421875, 0.587890625, -0.1036376953125, 1.2919921875, 0.53564453125, -0.07861328125, -0.89111328125, 1.5712890625, -1.1826171875, -0.06707763671875, -0.19140625, 0.254150390625, 1.2783203125, -0.131103515625, 0.09356689453125, -0.25830078125, -0.493408203125, 0.841796875, 0.46337890625, 0.71826171875, -2.322265625, -1.990234375, -0.98486328125, -0.36767578125, -0.80517578125, -0.27685546875, -0.302001953125, 1.359375, -0.262939453125, 0.9111328125, -0.1771240234375, 0.258056640625, 0.71484375, -0.78076171875, 0.1475830078125, 0.89306640625, 1.373046875, -0.5458984375, 0.491943359375, 0.1722412109375, 0.2132568359375, -0.430419921875, 0.90478515625, -0.642578125, -0.72900390625, 0.66162109375, 0.0731201171875, -0.93115234375, 1.2392578125, 0.79638671875, -0.0108489990234375, -0.810546875, -1.1962890625, 0.66845703125, -0.00360107421875, 0.149658203125, -0.12286376953125, 0.44775390625, -1.8984375, -0.29248046875, -0.370849609375, 1.0693359375, 1.0068359375, -2.1328125, -0.650390625, 0.84521484375, 1.796875, 1.1845703125, -0.5712890625, -0.1531982421875, -0.392333984375, -1.4716796875, -1.3818359375, 0.461181640625, 1.5244140625, -1.81640625, 0.55712890625, 0.390625, 1.0439453125, 1.4765625, 0.476806640625, 0.365478515625, -1.40625, 0.87255859375, 0.703125, 0.310791015625, 0.7744140625, 0.78271484375, 0.197998046875, -1.888671875, -0.17138671875, 1.03125, -0.354736328125, 0.73095703125, -1.1806640625, 0.055511474609375, 1.0263671875, 0.495849609375, 0.08935546875, 0.83837890625, 0.591796875, 0.72802734375, 0.10247802734375, -0.416748046875, -1.8759765625, -0.57666015625, 0.458740234375, 0.0706787109375, -0.638671875, -2.60546875, -0.8291015625, -0.74072265625, 2.216796875, 0.8525390625, 0.40625, -0.0125274658203125, -0.472900390625, 1.1728515625, 0.5537109375, -0.099853515625, 0.2322998046875, 0.144287109375, -1.1025390625, 0.57861328125, -1.16796875, -1.154296875, -0.2279052734375, 0.59375, 0.35498046875, -0.476318359375, 1.845703125, 0.5361328125, -0.471923828125, -0.1790771484375, -0.38671875, 1.5244140625, 0.32763671875, -0.35986328125, -2.015625, 1.3046875, 0.62939453125, 0.8818359375, -1.1748046875, 0.406494140625, -0.8154296875, 0.38818359375, -1.6513671875, -1.1611328125, 0.5009765625, 1.158203125, 1.427734375, 2.30078125, -1.2431640625, 1.1220703125, 1.314453125, -0.432861328125, -1.2236328125, -0.51953125, 0.430419921875, 0.5078125, -1.0322265625, -0.425048828125, 0.233154296875, -0.26611328125, -0.392333984375, 0.64013671875, -0.1990966796875, -1.611328125, -1.7705078125, -0.436279296875, -0.1881103515625, -0.62890625, 0.11553955078125, 1.8642578125, -0.68994140625, -0.01312255859375, -0.4873046875, 0.1617431640625, 0.32568359375, -1.3486328125, 0.6142578125, -2.337890625, 2.63671875, 0.59619140625, 0.052032470703125, -0.52783203125, -0.3466796875, 0.884765625, -1.1044921875, 0.06597900390625, 0.1527099609375, -0.7744140625, -0.1925048828125, 1.189453125, -0.91845703125, 1.7021484375, 1.0595703125, -0.828125, 1.6240234375, 0.6337890625, -0.69140625, 0.7314453125, -0.11199951171875, -0.97265625, -1.0078125, 0.489501953125, -0.80859375, 1.8681640625, 0.377685546875, 0.6064453125, -0.96728515625, 0.91943359375, -0.97900390625, -1.1748046875, -2.17578125, 0.68798828125, 0.454345703125, -0.1749267578125, 0.66552734375, -0.853515625, -1.6787109375, 0.66015625, -1.1796875, -1.5712890625, -1.091796875, 0.312744140625, -1.1474609375, 0.4541015625, 1.4111328125, -1.6279296875, -0.038665771484375, -1.2861328125, -0.2052001953125, -0.8037109375, -0.341796875, 0.7353515625, -0.21484375, -0.7919921875, 0.1097412109375, 0.9990234375, 0.5791015625, -0.189697265625, -0.79248046875, -1.29296875, -0.4208984375, 0.24755859375, -1.0244140625, -2.623046875, -1.9580078125, 1.1337890625, 0.07818603515625, -1.5634765625, 0.322265625, 0.734375, 2.7578125, 0.52392578125, 1.0419921875, -0.1011962890625, 0.24267578125, 0.96484375, 0.257080078125, -0.10711669921875, -0.3837890625, 0.6943359375, 0.1058349609375, -0.12939453125, 0.14453125, -0.19873046875, 1.8876953125, -0.1641845703125, -2.095703125, 0.06829833984375, -0.343994140625, -0.30859375, 1.724609375, 0.0225677490234375, 0.72216796875, -0.10986328125, 0.1883544921875, 0.658203125, 0.58349609375, 0.56494140625, 0.0018796920776367188, -0.56103515625, -0.54052734375, -0.33154296875, 0.63037109375, 1.1494140625, -0.75, -0.1868896484375, -0.806640625, 0.80712890625, 0.3974609375, 0.1826171875, -0.40087890625, 0.489990234375, 0.2105712890625, 1.125, 0.3798828125, -1.05859375, 1.5029296875, -1.2158203125, -0.301025390625, -0.775390625, -0.32763671875, 0.279296875, -0.61181640625, -0.301513671875, 0.845703125, 1.4658203125, -1.154296875, -0.03399658203125, 0.70263671875, -0.8193359375, 0.1939697265625, -0.92822265625, 0.35791015625, 1.2333984375, 1.595703125, 0.9951171875, 0.15478515625, -0.384521484375, -0.88916015625, 0.2313232421875, -0.7099609375, 1.3095703125, 0.60693359375, -0.0362548828125, -0.2037353515625, -0.9990234375, 0.8955078125, -0.332763671875, 0.59814453125, -0.97705078125, -1.25, 1.041015625, -1.4453125, -0.3505859375, 0.72509765625, -1.1455078125, 0.22998046875, 0.25927734375, -0.5361328125, 1.841796875, 1.3525390625, -0.0222320556640625, 0.487548828125, 0.1636962890625, 1.0322265625, -1.4140625, 1.1474609375, -0.10736083984375, 0.716796875, -0.943359375, -0.70458984375, -0.349365234375, -0.55615234375, -0.54345703125, 0.40283203125, 0.51708984375, -2.734375, -2.0546875, -1.2998046875, 0.4521484375, -0.7236328125, 0.239013671875, -0.339599609375, 0.221923828125, -0.2313232421875, 0.86083984375, -2.123046875, -1.0380859375, -3.27734375, -1.01171875, 1.17578125, 0.206298828125, 0.447021484375, -1.228515625, 0.68408203125, 0.08984375, 2.30859375, -0.454345703125, -0.47900390625, -0.76513671875, -0.86767578125, -1.7255859375, -0.775390625, -0.405517578125, 0.92041015625, -0.014434814453125, -1.6220703125, 0.53125, -1.328125, 1.8583984375, -0.5986328125, -0.1395263671875, -0.56494140625, 0.448974609375, -0.26513671875, -0.5673828125, -1.4482421875, 1.1142578125, -0.92724609375, -1.2333984375, -0.06561279296875, 1.3408203125, 0.330322265625, -1.0302734375, 1.052734375, 0.05523681640625, 0.2281494140625, -0.7626953125, 0.08856201171875, -0.410400390625, 1.0576171875, 0.32763671875, -0.404052734375, -1.0048828125, 0.3427734375, 1.1005859375, -0.9921875, -1.1884765625, 0.50146484375, 0.04217529296875, -0.462158203125, 0.779296875, -0.505859375, -0.2061767578125, 0.0175018310546875, 0.84814453125, 1.0634765625, -1.7109375, 0.142333984375, -0.364013671875, 0.71728515625, -0.91259765625, -0.2408447265625, 0.28515625, -1.4208984375, 2.8203125, -0.477783203125, -0.8134765625, -1.6904296875, -0.399658203125, 0.65087890625, 1.1650390625, 0.6611328125, 2.5625, -0.56396484375, -0.46435546875, 0.0697021484375, 0.65380859375, 0.261962890625, 1.650390625, -1.4365234375, 0.89208984375, 0.6416015625, 0.78564453125, -0.8515625, 0.11138916015625, -1.4384765625, -1.275390625, -0.67578125, 1.650390625, -1.6904296875, 0.3720703125, -0.08233642578125, 1.666015625, -0.416748046875, 1.9365234375, -0.336669921875, 0.84716796875, 0.09912109375, 0.498779296875, -1.9775390625, -0.6162109375, -0.0019893646240234375, 0.15380859375, 0.323486328125, 0.3603515625, 0.87158203125, -0.23876953125, 1.3369140625, 0.29541015625, 2.130859375, 1.3515625, -1.1259765625, -1.015625, -0.312744140625, -0.2117919921875, 0.6953125, 0.0106048583984375, -0.9287109375, 1.0791015625, 0.95751953125, 0.394775390625, 0.49609375, -1.53125, 0.71435546875, -1.916015625, 0.84912109375, -0.271728515625, -1.9052734375, -1.642578125, 0.7734375, 1.0380859375, 0.376220703125, -0.2447509765625, 0.97607421875, -0.85205078125, 1.6064453125, 0.69970703125, -1.833984375, 0.052581787109375, 0.5107421875, -0.98291015625, 0.53564453125, -0.3515625, -0.72802734375, 0.6279296875, -0.06488037109375, 0.52294921875, -0.424072265625, 1.2490234375, -0.41357421875, 1.544921875, -1.3798828125, 0.40966796875, -1.583984375, -0.53564453125, -0.7294921875, -2.34375, 0.12103271484375, 0.66162109375, 0.4580078125, 1.3251953125, -0.45361328125, -1.07421875, -0.8388671875, -0.459716796875, -0.74853515625, -0.30615234375, 0.4287109375, -1.21484375, -1.1572265625, -0.62158203125, 2.064453125, 1.474609375, -1.4482421875, 0.40771484375, 0.359130859375, 0.94970703125, -0.5029296875, 0.1385498046875, -0.7900390625, -0.09320068359375, 0.1282958984375, -0.48974609375, 0.5029296875, -0.97021484375, 0.5234375, -2.119140625, -0.481201171875, 1.849609375, 0.2017822265625, -1.39453125, -0.9072265625, -2.328125, 0.4150390625, -0.73828125, -0.144775390625, 0.5400390625, -0.55224609375, -0.8076171875, 1.0107421875, -0.0606689453125, -0.92724609375, 0.38330078125, 2.533203125, 1.1689453125, 0.342529296875, -0.1610107421875, 1.298828125, -1.453125, -0.85546875, -1.1142578125, -1.078125, 0.210693359375, 1.904296875, 0.08514404296875, 0.81201171875, -0.314453125, 1.3232421875, 0.2108154296875, 2.34765625, -0.611328125, 0.7275390625, 0.78955078125, -0.51123046875, 1.5859375, -1.1533203125, 0.9970703125, 0.63623046875, 1.4501953125, 2.021484375, 1.013671875, 0.5205078125, 0.257080078125, -1.1787109375, 0.69384765625, -0.195068359375, -0.425537109375, -1.720703125, -0.25830078125, 0.2021484375, -0.65234375, -0.74169921875, -0.471923828125, -1.076171875, 1.419921875, 0.98583984375, 1.4208984375, 0.7333984375, -0.468505859375, 0.68115234375, 0.7109375, -0.424072265625, 0.264892578125, 0.26123046875, 1.013671875, -0.142578125, -0.81201171875, -0.9072265625, 0.3984375, 1.0205078125, -0.81640625, -0.08135986328125, 0.1142578125, -1.90234375, 0.07659912109375, -0.52734375, -0.283203125, -1.7900390625, 0.65869140625, 0.6142578125, -0.60498046875, 1.1689453125, -1.498046875, -0.478515625, 0.83447265625, 0.11468505859375, -0.459716796875, 1.138671875, 0.82275390625, -0.69287109375, -0.21923828125, -0.2454833984375, -1.01953125, 1.330078125, -1.5283203125, -0.468505859375, 0.397705078125, -0.7734375, 0.32763671875, 1.9189453125, -1.38671875, 1.2255859375, -0.3974609375, 0.9423828125, -1.5927734375, 0.68115234375, 0.6474609375, -0.04052734375, -0.46630859375, -0.00821685791015625, -0.281494140625, 0.444580078125, -0.038177490234375, -0.7685546875, -1.0791015625, 0.270263671875, 0.499755859375, -1.16796875, 0.8701171875, 0.5732421875, 1.2822265625, -0.99462890625, -0.402587890625, 0.6787109375, 0.255859375, 0.4365234375, -0.1312255859375, 1.0556640625, -1.0810546875, 0.72021484375, -0.7578125, 0.7060546875, 0.58056640625, 0.2161865234375, 0.480712890625, -0.7265625, 0.24658203125, -0.07257080078125, 0.2025146484375, 0.33251953125, 1.3984375, -0.80810546875, 0.646484375, -0.91357421875, -1.9658203125, 1.0341796875, -1.345703125, -0.8017578125, 2.40625, 1.431640625, -0.468994140625, -1.052734375, -1.26171875, 0.388671875, 0.611328125, 0.1248779296875, -0.456787109375, 0.413818359375, -1.8408203125, -0.247802734375, 0.6669921875, -0.11541748046875, -0.91162109375, -2.146484375, 1.08984375, 0.328125, 0.116943359375, 0.9912109375, -0.361083984375, 0.6357421875, 1.666015625, -0.72265625, 0.751953125, 0.8359375, -0.73388671875, -0.6337890625, -1.2646484375, 1.3017578125, 0.2314453125, -0.640625, -0.72412109375, -0.435791015625, 1.484375, 0.77392578125, -1.4873046875, 0.56298828125, 1.3818359375, -0.46533203125, 0.65673828125, -1.2080078125, 0.990234375, -0.67138671875, -0.53369140625, -0.01450347900390625, -0.228515625, -1.8427734375, 0.021881103515625, -0.40087890625, 0.705078125, 0.54638671875, -1.033203125, -0.7080078125, -1.16015625, 0.89697265625, 2.5703125, -0.984375, 0.381591796875, 0.346923828125, -1.5537109375, 0.6259765625, 0.1630859375, 1.0849609375, -2.310546875, 0.90380859375, 1.3681640625, 0.1328125, -0.6953125, 0.10015869140625, -0.92431640625, -0.31201171875, -1.060546875, -0.1492919921875, 1.2568359375, 0.57861328125, -1.421875, 1.96875, 1.5498046875, 0.74560546875, 0.62109375, -0.84326171875, -1.0439453125, -0.7802734375, -0.2183837890625, -0.50439453125, -0.86865234375, -0.73095703125, 0.81396484375, 1.0634765625, -0.818359375, 0.87158203125, 0.002964019775390625, 1.1201171875, -1.50390625, -1.9990234375, -1.9501953125, -0.79736328125, -0.1793212890625, 0.544921875, -0.8759765625, -1.2421875, 1.5029296875, -0.0902099609375, -0.72119140625, -0.06549072265625, -2.306640625, -0.57763671875, -0.0155792236328125, -1.26171875, 0.9677734375, -0.896484375, -0.359375, -0.25244140625, -0.29150390625, -2.634765625, 0.6259765625, -1.236328125, -0.40576171875, 1.5908203125, 1.958984375, -0.78076171875, 1.556640625, 0.225341796875, 0.1717529296875, 0.4814453125, -0.257080078125, 0.01190185546875, 0.7021484375, 0.301513671875, -0.869140625, 0.55419921875, 3.541015625, -0.2125244140625, -0.487548828125, 0.89501953125, 0.1739501953125, -0.37548828125, 0.31494140625, -1.6083984375, -0.2283935546875, -0.798828125, -1.505859375, 0.7666015625, -2.029296875, -0.20068359375, 0.055084228515625, -0.77099609375, 1.3466796875, -0.472900390625, -0.320068359375, 1.810546875, 1.37109375, 0.2427978515625, 0.1900634765625, 0.97998046875, -1.892578125, -1.2314453125, -0.253173828125, -1.2412109375, -1.1953125, 0.58837890625, 0.81298828125, 1.9521484375, 1.5830078125, 0.96630859375, -0.028411865234375, -0.6650390625, 0.09912109375, 0.513671875, -0.771484375, 0.28076171875, 1.24609375, -0.1383056640625, -0.99267578125, -0.034759521484375, 0.5732421875, -0.6865234375, -1.3798828125, -0.1851806640625, 1.3681640625, 0.91845703125, -0.2418212890625, 0.364013671875, 0.0765380859375, 0.36669921875, 0.71337890625, -0.46630859375, -1.556640625, 1.142578125, -1.302734375, -0.418212890625, 0.6533203125, -1.3017578125, -0.199462890625, -1.12890625, 1.078125, 1.072265625, 2.75, 2.80078125, 0.9541015625, -1.259765625, 0.334228515625, -0.44189453125, -0.72412109375, -1.015625, -0.55712890625, -1.3798828125, 1.4228515625, 0.6728515625, -1.4423828125, 1.099609375, -1.443359375, 0.712890625, -1.5732421875, -1.490234375, -2.029296875, 1.2158203125, 0.57568359375, -0.2357177734375, -0.1322021484375, -0.355712890625, -1.751953125, -0.177001953125, -0.73291015625, -0.486083984375, 0.28515625, -1.5576171875, -0.5087890625, -0.410888671875, -1.0771484375, 1.130859375, -1.1748046875, 2.31640625, -0.364013671875, 1.2392578125, -2.08203125, -0.289306640625, -0.7373046875, -0.1165771484375, 1.5439453125, 0.37353515625, 0.7529296875, 0.9873046875, 1.7333984375, 1.09375, 0.99072265625, 1.7890625, -0.0458984375, -0.73095703125, 0.06195068359375, 0.63720703125, 0.80029296875, -2.501953125, -0.61083984375, -0.00020563602447509766, -1.4765625, 1.7099609375, 1.1279296875, 1.4609375, 1.0537109375, -2.072265625, -0.357666015625, -1.05859375, 1.724609375, 1.552734375, 0.189453125, 0.6318359375, -0.288330078125, 0.81884765625, 0.093505859375, 0.2587890625, -0.82080078125, 0.473388671875, -0.0021820068359375, 0.40234375, -0.9794921875, 0.53662109375, 0.0750732421875, 1.275390625, -0.343505859375, 1.876953125, 1.0966796875, 1.0263671875, -0.29833984375, 1.4423828125, 0.01078033447265625, 1.4921875, -1.5380859375, 0.83349609375, -1.494140625, 0.73095703125, 0.779296875, -0.466796875, -0.2066650390625, 2.599609375, -1.8486328125, -0.2235107421875, 0.8798828125, -0.58740234375, 0.366455078125, -1.0283203125, -1.4931640625, 0.72998046875, -0.8134765625, -2.017578125, 1.3603515625, 1.943359375, -1.4453125, 2.470703125, 0.62646484375, 1.2578125, -1.3017578125, 1.1337890625, -0.0960693359375, 0.65576171875, 2.830078125, -0.83642578125, 0.07781982421875, 1.634765625, -0.348388671875, 0.60107421875, 0.18359375, -0.40478515625, 0.685546875, 1.0703125, 0.51416015625, -0.51953125, 0.370849609375, 0.289306640625, 0.1986083984375, 0.1234130859375, -0.59716796875, 0.72314453125, -1.060546875, -0.56494140625, 1.7607421875, -0.425048828125, 1.9501953125, 0.098388671875, -1.169921875, -1.291015625, -0.09185791015625, -1.33984375, 0.1158447265625, 1.310546875, -0.626953125, -0.44140625, -1.4208984375, -0.68701171875, 1.388671875, 0.231689453125, 1.0244140625, 0.53955078125, 0.74169921875, 0.822265625, 0.43310546875, -0.11553955078125, -1.53515625, -0.29150390625, -1.2197265625, -1.41796875, -0.603515625, -1.525390625, 0.2088623046875, 1.228515625, 0.479248046875, 0.04400634765625, 0.8984375, -0.68603515625, 0.102294921875, 0.296142578125, 0.1370849609375, -0.01727294921875, -0.533203125, 2.169921875, -0.865234375, 1.6826171875, -2.439453125, 0.2037353515625, 0.7294921875, 0.82373046875, 1.634765625, 1.2841796875, 0.62353515625, -0.1798095703125, 0.1043701171875, -0.732421875, 0.72900390625, 0.408203125, 0.88720703125, 0.430908203125, 0.81689453125, -0.2462158203125, 0.41845703125, 2.173828125, -0.15673828125, 0.6943359375, -0.2357177734375, -1.439453125, 0.8916015625, -0.6669921875, 0.389892578125, -0.72900390625, 1.294921875, 0.12139892578125, 0.141845703125, 0.04498291015625, 2.73828125, -0.2369384765625, -1.998046875, -0.43212890625, 1.201171875, -0.2210693359375, -0.841796875, 0.951171875, 0.350830078125, -1.1416015625, 0.56640625, -0.650390625, 0.2491455078125, 0.165283203125, 0.22216796875, 0.187744140625, -1.1142578125, -1.7451171875, 0.2166748046875, -0.49951171875, -0.0251617431640625, 1.5263671875, -0.1956787109375, -1.4150390625, -0.393798828125, 0.48046875, -0.2080078125, -0.288330078125, -2.4375, 0.38623046875, -1.1494140625, -0.2392578125, 0.3447265625, 2.0234375, -0.356689453125, 2.544921875, -0.515625, -0.05322265625, -1.435546875, 2.015625, -0.6328125, -1.15234375, -0.50537109375, -0.86083984375, -1.146484375, -0.18994140625, 0.0231170654296875, 0.3955078125, 0.56689453125, -0.65869140625, -0.42236328125, -0.09893798828125, -0.11688232421875, 2.2890625, 2.3359375, -0.4462890625, -0.1300048828125, 1.62890625, 1.1279296875, -0.5693359375, -0.61083984375, -1.4716796875, 0.857421875, 1.1064453125, -2.080078125, 1.380859375, -1.5966796875, -0.0004870891571044922, -1.4775390625, 0.50830078125, 1.5234375, -0.361083984375, -1.0439453125, -0.99658203125, 0.5078125, -1.6484375, 0.39599609375, 1.11328125, 1.4775390625, -1.6982421875, 0.65625, -0.036041259765625, 0.8974609375, -0.80078125, 2.353515625, -1.671875, -1.8525390625, -0.56396484375, -0.78271484375, -1.7666015625, 0.755859375, -1.8125, 0.978515625, 0.360595703125, -0.78515625, -0.26953125, 0.76953125, 0.93994140625, -1.3896484375, 0.4140625, -0.1802978515625, -1.28515625, 1.3369140625, 0.541015625, -1.2646484375, 0.206298828125, -0.62890625, 0.371337890625, -2.703125, -0.393798828125, -1.65625, 1.0498046875, 0.00872802734375, 0.40966796875, -0.2154541015625, -1.298828125, -1.2548828125, 0.325927734375, -0.9482421875, 1.55078125, -1.8251953125, 1.1123046875, 0.70068359375, 0.486572265625, -0.63671875, 0.09234619140625, -0.388427734375, -2.076171875, -0.37744140625, 2.123046875, 0.44580078125, -1.4033203125, -0.9208984375, -0.671875, 0.07135009765625, -0.6826171875, -1.4755859375, -0.30224609375, -0.421630859375, 0.39013671875, -0.423095703125, 1.8154296875, -1.33984375, -0.07763671875, -0.376708984375, 0.220947265625, 0.0011720657348632812, 0.17138671875, -1.130859375, -1.6708984375, 0.9794921875, -1.244140625, -0.168701171875, 0.0188751220703125, -1.720703125, -1.2177734375, 0.0675048828125, 1.0, 2.150390625, -0.383544921875, -0.294189453125, 0.546875, 1.611328125, -2.611328125, -0.6474609375, -0.43896484375, 0.230224609375, -0.8984375, -0.81640625, -0.82666015625, 0.96728515625, 0.3251953125, 0.0760498046875, -0.90869140625, 0.74755859375, 0.429443359375, -0.68994140625, 0.382080078125, -2.46875, -0.78271484375, -0.2900390625, -1.125, 1.296875, -0.876953125, -0.06170654296875, -0.560546875, -0.34912109375, -0.318115234375, -1.6015625, -0.4697265625, 0.292724609375, 0.00016379356384277344, -1.8115234375, -0.9169921875, 0.6142578125, 0.2437744140625, -0.0006999969482421875, 1.2109375, 0.5869140625, -0.277587890625, 0.33447265625, 0.9248046875, -1.4013671875, -0.494140625, -0.160888671875, 1.29296875, 1.3134765625, 0.156982421875, 0.39794921875, 0.1597900390625, -0.4287109375, 0.8642578125, 0.1859130859375, 0.87890625, -0.041656494140625, -0.9365234375, -1.150390625, 1.18359375, -0.050628662109375, 0.1285400390625, 0.556640625, 1.2021484375, 0.7744140625, -0.828125, -0.779296875, 1.0673828125, 0.9765625, -0.348876953125, -0.92919921875, -0.034698486328125, -0.97314453125, 0.1279296875, -1.8671875, -2.298828125, -2.384765625, -0.49951171875, -0.57421875, 0.037445068359375, -0.05474853515625, 0.169677734375, -0.947265625, 0.806640625, 1.373046875, -0.2978515625, 2.078125, 0.6298828125, -0.0201416015625, 0.8642578125, 0.5224609375, 0.53564453125, -0.74267578125, 0.0419921875, -0.01666259765625, -0.74365234375, 0.353515625, -1.5546875, -0.1976318359375, -1.09765625, 0.01081085205078125, 0.96630859375, -0.56103515625, 0.485595703125, 0.296875, 0.244873046875, 0.1636962890625, -0.52392578125, -1.171875, -0.9501953125, -0.91162109375, -0.37353515625, -1.375, 1.7666015625, 0.837890625, 2.9765625, 0.6806640625, 0.192626953125, 0.488037109375, 1.041015625, 1.3974609375, 1.03125, -0.451416015625, 0.4912109375, 1.1884765625, 0.7373046875, -0.52685546875, -0.7001953125, -1.1591796875, -1.203125, -0.85009765625, -1.1572265625, -1.9501953125, -0.1229248046875, -0.227783203125, 1.0537109375, 0.7998046875, 0.62939453125, -0.0631103515625, 0.244384765625, -0.9833984375, 1.2421875, 0.29150390625, 0.114990234375, -0.22607421875, -0.408203125, -0.64404296875, -1.37109375, 1.0654296875, -0.90869140625, -0.3271484375, -1.8017578125, -1.5810546875, -0.87548828125, 0.77880859375, -1.6396484375, 0.47412109375, -0.2265625, -0.40771484375, 0.0219268798828125, -1.453125, -0.3095703125, 2.04296875, 0.0679931640625, -1.41796875, -1.4892578125, 0.9736328125, 1.2041015625, -0.92431640625, 0.9365234375, 0.6396484375, 0.7939453125, 0.673828125, 3.033203125, -0.2919921875, -0.2066650390625, 0.204345703125, -0.67626953125, -1.3212890625, -1.259765625, 0.21875, -0.2073974609375, 1.1728515625, -1.3623046875, -0.371337890625, -1.12890625, -1.1865234375, 2.494140625, -0.84130859375, 0.134033203125, -1.009765625, 1.5888671875, -1.154296875, 0.46630859375, -0.5126953125, -0.64013671875, -1.349609375, 0.70654296875, 0.21142578125, -0.182373046875, 2.04296875, 0.7060546875, -0.430908203125, 0.533203125, 0.454833984375, 0.27685546875, 0.51611328125, -2.12890625, -0.61669921875, 0.923828125, -1.25, 1.63671875, -1.537109375, -2.791015625, -0.077392578125, 1.5380859375, -1.005859375, 1.779296875, 1.20703125, -0.6591796875, -0.85107421875, -0.89404296875, -0.86572265625, -1.3759765625, -0.0576171875, 0.1265869140625, -0.314453125, -1.1455078125, 0.0012121200561523438, -0.76953125, -0.39111328125, -1.0498046875, -1.544921875, 0.16357421875, -0.1927490234375, -1.38671875, -0.9892578125, 0.377685546875, 1.8447265625, -1.28515625, 2.33984375, 0.4580078125, 0.3349609375, -1.4765625, -1.314453125, 0.6708984375, -1.3984375, -1.29296875, 0.0943603515625, -1.310546875, 1.4033203125, 0.390380859375, 0.56103515625, -0.6474609375, -0.0439453125, 1.91796875, -0.9921875, -0.52880859375, -0.994140625, -0.14208984375, 1.078125, 1.154296875, -0.61669921875, 0.378173828125, -0.59033203125, -0.426513671875, 0.64208984375, -1.1845703125, 1.0107421875, 0.984375, 1.419921875, -1.341796875, -1.5927734375, 1.12109375, -0.10858154296875, 0.1680908203125, 0.525390625, -1.1806640625, -0.60302734375, 1.658203125, -1.5341796875, -2.275390625, -0.4033203125, 0.328369140625, -0.68212890625, 0.431396484375, -0.98486328125, -0.5546875, 0.77734375, 1.0771484375, -0.2012939453125, -0.02587890625, -1.1005859375, -0.0153350830078125, 1.00390625, -0.681640625, 0.341064453125, -1.693359375, -1.080078125, -0.796875, 0.82421875, 1.1484375, 0.634765625, -0.52490234375, 1.744140625, 0.333251953125, 0.97216796875, 0.9326171875, 1.7548828125, 0.239501953125, -0.21533203125, 1.033203125, -0.26416015625, -1.224609375, -0.748046875, -0.031402587890625, 1.064453125, 2.30859375, 0.9013671875, -0.422607421875, 0.66552734375, 0.880859375, 0.330810546875, 0.2320556640625, 0.06982421875, -1.0224609375, 1.0068359375, -0.931640625, 0.488037109375, -1.5751953125, 0.470458984375, -1.0205078125, -1.380859375, -0.195068359375, -0.1837158203125, 1.4189453125, -1.0263671875, 1.759765625, 0.3037109375, -0.59326171875, -0.54248046875, 0.2344970703125, 0.120361328125, -0.8583984375, 1.9208984375, -1.173828125, 0.326904296875, 0.035736083984375, -0.433837890625, 0.268798828125, 0.328369140625, 0.0089263916015625, 0.125732421875, -0.2054443359375, -0.232177734375, -1.8427734375, 1.9990234375, 0.6923828125, -1.1328125, -0.156005859375, 2.900390625, 2.783203125, -0.75830078125, -0.2685546875, -0.455322265625, 1.5830078125, 0.6630859375, 1.1650390625, -0.7548828125, 0.74755859375, -1.1962890625, -0.5927734375, 1.0048828125, -1.326171875, 1.2197265625, -1.5419921875, -0.417724609375, -1.4609375, 1.3154296875, -0.495361328125, 1.126953125, 2.509765625, 0.379638671875, 0.55859375, 1.0, 0.00501251220703125, -0.061767578125, -0.254638671875, 0.349609375, -0.7265625, -0.1790771484375, 0.247802734375, 0.0221405029296875, 0.9052734375, -0.94287109375, -0.058563232421875, -0.50244140625, 0.62744140625, 0.440673828125, -0.11944580078125, 0.0193023681640625, -0.9287109375, 1.583984375, 2.3046875, -0.0285797119140625, -0.1876220703125, 0.64306640625, 1.134765625, -2.111328125, 0.06146240234375, 0.8369140625, -0.9462890625, 2.083984375, -2.537109375, -0.07977294921875, -0.7294921875, 1.611328125, 1.388671875, 0.6328125, 1.7626953125, 0.90576171875, 0.7255859375, 0.62060546875, -0.3017578125, -0.0293426513671875, 0.50537109375, 0.9892578125, 1.017578125, -1.779296875, 1.2431640625, -0.025146484375, 0.3271484375, -1.9638671875, -2.125, 0.042083740234375, 0.50048828125, -0.7578125, -0.72607421875, 1.3134765625, 0.255859375, -1.3134765625, -1.3056640625, -0.68603515625, 1.0439453125, 0.470703125, -0.25634765625, -1.12109375, -1.37109375, 1.12109375, 0.92822265625, -1.9609375, 2.033203125, 2.298828125, -0.01424407958984375, 0.180908203125, 0.133056640625, 1.2890625, -2.34765625, 0.415283203125, 1.04296875, -0.170166015625, 1.0419921875, -0.48974609375, 0.1524658203125, 0.045135498046875, -0.82373046875, 1.5478515625, -0.6259765625, 1.3427734375, -0.355712890625, 1.0751953125, 0.736328125, 0.41748046875, 0.255126953125, -0.205810546875, -0.1668701171875, -1.1494140625, -0.0989990234375, -0.1610107421875, 0.0955810546875, -0.1866455078125, 0.1302490234375, -1.5302734375, -0.2568359375, 0.01065826416015625, 0.19921875, -0.23046875, 1.0, -1.0859375, 0.397216796875, -1.0244140625, 0.91259765625, -0.16162109375, 1.46875, 0.266845703125, -1.5498046875, -2.20703125, 1.3740234375, -0.93896484375, -0.55908203125, -0.481689453125, 0.515625, -0.411376953125, -3.044921875, -2.578125, -1.220703125, -0.5654296875, -1.4111328125, 0.56494140625, 0.60205078125, -0.1370849609375, -0.00823974609375, 0.54248046875, 1.73046875, 0.6923828125, 1.5478515625, 1.09375, -1.271484375, 1.1650390625, -0.052703857421875, 0.9306640625, 0.371337890625, -1.6669921875, 0.8330078125, 1.205078125, 0.87353515625, 0.042236328125, -0.8818359375, -0.58642578125, 0.49267578125, 0.53662109375, -1.35546875, -0.5986328125, -0.62060546875, 0.1689453125, -0.1878662109375, -1.7158203125, 0.814453125, 0.2181396484375, 2.015625, 0.62939453125, -0.56494140625, -0.4453125, -0.806640625, 0.393798828125, -0.297119140625, 0.316162109375, 0.460693359375, -0.19580078125, -0.970703125, -0.489013671875, -1.166015625, -0.61474609375, 2.115234375, -0.7900390625, -0.5400390625, 0.33349609375, 1.56640625, 1.6201171875, -0.342041015625, -0.82958984375, -0.48486328125, 0.57373046875, -1.9423828125, -1.376953125, -0.1517333984375, -0.2423095703125, -0.411865234375, 1.017578125, -1.11328125, 1.0009765625, 0.7646484375, 0.72607421875, -1.22265625, -0.673828125, -0.8671875, -0.70361328125, 0.042449951171875, -1.73828125, 0.65869140625, -0.53369140625, -1.0517578125, -0.7216796875, 0.368408203125, -0.6572265625, -0.1666259765625, -1.4482421875, -0.468505859375, -1.7578125, 0.325439453125, 1.1484375, 0.488525390625, -0.18408203125, 0.5625, -0.441650390625, -0.83984375, 0.307373046875, 0.322998046875, 0.08331298828125, -0.5341796875, -1.125, 0.00943756103515625, 0.00733184814453125, 0.38525390625, 1.71484375, 0.01340484619140625, -1.861328125, 0.3115234375, 0.1407470703125, -0.16455078125, -0.419189453125, -0.211181640625, 0.1480712890625, 0.447509765625, -1.1875, 0.296142578125, -0.2978515625, -0.603515625, -0.0792236328125, -1.638671875, 1.04296875, -1.3203125, -0.27685546875, -1.0205078125, -0.8408203125, 0.80419921875, -1.427734375, -0.72216796875, -0.348876953125, -1.7802734375, -0.007904052734375, 0.54931640625, 1.423828125, -1.259765625, 0.7958984375, 2.248046875, -0.32275390625, -0.04229736328125, 0.88671875, 0.74609375, 0.70751953125, -0.21044921875, 0.46875, -0.432373046875, -0.9345703125, -0.337890625, 0.5302734375, 0.6298828125, -2.59375, 1.353515625, 0.39697265625, 0.13671875, 1.7890625, 0.75390625, 2.56640625, 0.376220703125, -0.353271484375, -1.7626953125, -0.2261962890625, -1.212890625, -1.8017578125, -0.211669921875, -1.599609375, -0.345947265625, 0.833984375, 0.281005859375, 0.366455078125, -0.412841796875, 1.4140625, -1.02734375, -0.3447265625, 0.46435546875, -1.28125, 2.16796875, 1.072265625, -0.56396484375, 0.71875, 0.404296875, 0.379150390625, 0.6435546875, -0.08544921875, 0.73779296875, 2.4140625, 0.00893402099609375, -2.41796875, 1.408203125, -0.297119140625, -0.477294921875, 1.08203125, -1.55859375, -0.81591796875, 1.572265625, 0.296875, -1.1640625, -0.1619873046875, -0.61572265625, 1.1376953125, 1.0517578125, -0.623046875, 0.01324462890625, -0.53369140625, 0.0355224609375, 0.62890625, -0.26953125, -0.95068359375, 0.0343017578125, -0.5283203125, 0.97998046875, 0.468017578125, -2.3984375, 1.7734375, -0.74072265625, -1.453125, -1.044921875, 0.7724609375, 1.529296875, -0.6962890625, 0.8828125, -1.5439453125, 0.8115234375, 0.84375, 0.12841796875, -0.748046875, 0.1630859375, 0.67333984375, -0.7431640625, -0.81689453125, 1.1630859375, 0.42431640625, -0.2412109375, 0.51806640625, 1.1201171875, 0.63232421875, 0.06353759765625, 0.4521484375, -1.7578125, 2.78125, 0.186279296875, 0.037139892578125, -1.33984375, 0.50244140625, 0.68310546875, 0.89208984375, 0.033233642578125, -0.66650390625, 0.7431640625, -0.369140625, 0.68603515625, -0.50732421875, -0.54443359375, -1.0625, 0.133056640625, -0.53125, 0.79931640625, 0.3447265625, -0.84912109375, -0.62158203125, 0.11431884765625, -0.79443359375, -0.5361328125, 1.322265625, -0.34033203125, -1.1669921875, -0.48681640625, -1.421875, 1.5087890625, -0.12054443359375, -1.171875, -0.73876953125, -0.254150390625, -0.480712890625, -0.161376953125, 0.263427734375, -0.10296630859375, -0.39013671875, -1.306640625, -0.276123046875, -0.10809326171875, 0.54736328125, -0.438232421875, 0.4052734375, -1.193359375, -0.58349609375, 0.31689453125, -0.68359375, 0.2381591796875, 1.11328125, 0.426513671875, -0.1705322265625, -0.0565185546875, 0.82763671875, -0.1888427734375, 1.4306640625, -1.1533203125, 0.2486572265625, -0.451171875, -0.33349609375, -0.3984375, 0.1356201171875, -2.091796875, -0.52587890625, 0.327392578125, -0.55615234375, -0.222900390625, 0.83642578125, 0.583984375, 0.467041015625, 0.92236328125, -0.239501953125, 0.568359375, -0.316162109375, 0.07110595703125, 0.51953125, 0.6259765625, -0.5322265625, 0.54833984375, 0.032989501953125, -0.9462890625, -2.05859375, 1.7939453125, -0.336669921875, 0.306396484375, 0.6025390625, -1.9775390625, 0.100830078125, 0.69970703125, -0.125732421875, 0.2490234375, 0.82275390625, 0.2269287109375, 0.8857421875, -1.0224609375, -0.77978515625, 1.5478515625, -0.277587890625, -0.187255859375, 0.141845703125, -0.1632080078125, 0.043060302734375, -0.11248779296875, 0.251953125, 0.07269287109375, -0.33984375, 0.9755859375, -1.0654296875, -1.3798828125, 1.1123046875, 0.4228515625, -0.01393890380859375, 0.63720703125, 1.4990234375, 0.446533203125, 0.5673828125, 1.333984375, 0.49462890625, -0.3720703125, -0.1595458984375, 0.70849609375, -0.40869140625, -0.284423828125, 0.78955078125, 1.033203125, 0.95703125, -0.3232421875, 1.189453125, 2.296875, 0.78564453125, 0.99169921875, 0.96728515625, -0.69384765625, -0.93896484375, 0.1834716796875, 0.1739501953125, -0.27099609375, -0.1727294921875, -0.66259765625, 1.2783203125, 0.052337646484375, -1.4921875, 0.026458740234375, 1.2587890625, 1.3623046875, -0.301513671875, 1.0380859375, -0.2271728515625, 1.029296875, -0.0867919921875, -0.6953125, 0.210205078125, -0.8447265625, -0.27001953125, 0.0242919921875, 0.529296875, 1.5888671875, -0.7822265625, 0.043731689453125, 0.9833984375, 1.7041015625, 1.234375, -0.2275390625, -1.494140625, -1.12109375, 0.1451416015625, -0.6357421875, -0.049591064453125, -0.8916015625, -1.345703125, -0.56201171875, -0.7451171875, -0.12042236328125, -0.90478515625, -0.458740234375, 1.70703125, 0.4873046875, 1.390625, 0.50146484375, -0.86328125, -0.1746826171875, -1.0576171875, -0.2459716796875, -1.068359375, 0.057769775390625, 0.438232421875, -0.68408203125, 1.42578125, -0.5517578125, -0.58935546875, -0.6328125, 1.318359375, -1.8798828125, -0.9638671875, -0.60986328125, 0.377197265625, 0.685546875, -1.1982421875, 0.08740234375, 1.076171875, 0.27587890625, 0.66552734375, -0.3291015625, 2.142578125, 1.060546875, 0.368896484375, 1.283203125, -1.5009765625, 1.8564453125, -0.94775390625, -0.1300048828125, -1.724609375, -0.75927734375, 1.115234375, 0.11407470703125, -0.88916015625, 0.397705078125, -0.70166015625, -0.58154296875, 0.08392333984375, 0.2235107421875, -0.87255859375, 0.75634765625, -1.6240234375, 0.4169921875, 0.201904296875, -1.779296875, -1.07421875, 0.06488037109375, -1.02734375, 2.755859375, 0.83251953125, 0.471435546875, 0.08697509765625, 1.1923828125, 0.5546875, -0.50927734375, 0.63720703125, -0.318603515625, 0.06146240234375, 0.88720703125, 0.68212890625, -0.363525390625, -1.1240234375, 0.07440185546875, 0.962890625, -1.2685546875, 0.015533447265625, 0.7265625, -0.2900390625, -1.6591796875, -1.0, 0.490478515625, 1.50390625, -0.28955078125, 1.208984375, 0.75439453125, -0.65966796875, 1.078125, -0.39453125, -2.533203125, 0.453369140625, -0.276611328125, 1.4306640625, 1.1123046875, 0.453125, 0.61328125, 1.966796875, 1.2080078125, 0.06781005859375, -1.330078125, 1.0654296875, 3.1953125, -1.474609375, -0.213134765625, 0.2313232421875, 1.287109375, 0.35888671875, -0.97802734375, -0.7119140625, 1.841796875, 1.8076171875, -1.1123046875, -2.005859375, -1.1806640625, -1.3759765625, 0.107666015625, -0.1483154296875, -0.482666015625, 1.0927734375, 0.1624755859375, 1.3291015625, -0.255615234375, -0.10064697265625, 0.261474609375, -0.483154296875, 1.849609375, -1.8203125, 0.4697265625, 0.300537109375, 0.454833984375, -0.97900390625, 1.634765625, 0.90380859375, 0.0716552734375, 1.49609375, 0.011871337890625, 2.015625, -1.5390625, 0.26123046875, 0.98876953125, -0.40869140625, -0.413330078125, 0.2061767578125, -1.2666015625, 1.1484375, -1.0283203125, 0.051605224609375, 1.1259765625, 1.876953125, -1.5322265625, -1.2802734375, -0.1558837890625, 1.541015625, -0.98095703125, -0.44921875, 0.53271484375, 0.295166015625, 0.41455078125, -0.01561737060546875, 0.63427734375, -1.2861328125, 0.54052734375, -0.64501953125, 1.4306640625, 0.0272979736328125, -1.1953125, 0.315185546875, -0.1102294921875, 0.89111328125, 0.292724609375, -0.72412109375, 1.001953125, -0.53125, 0.10565185546875, 1.83984375, 0.94140625, -0.08673095703125, -0.58056640625, 0.188720703125, -0.1810302734375, -0.58544921875, 0.357666015625, 1.2470703125, -0.7177734375, -0.27734375, -1.142578125, 0.93212890625, -0.33935546875, -1.91015625, -0.6064453125, 0.98095703125, 0.0279083251953125, 0.055206298828125, 1.5703125, -0.50341796875, 0.17822265625, -1.0068359375, -0.457275390625, 0.78955078125, 0.05938720703125, 0.421630859375, -0.32861328125, 2.056640625, 1.296875, -0.23681640625, -1.4306640625, -1.6962890625, -0.3125, -0.276611328125, -0.474853515625, 0.63037109375, -0.04840087890625, -1.314453125, -1.9609375, -1.1171875, 1.5380859375, 0.583984375, -0.6396484375, 0.08465576171875, -0.37353515625, -0.0845947265625, 0.400390625, 1.73046875, 0.296630859375, 0.0885009765625, -0.42822265625, 0.313232421875, 1.228515625, 0.00445556640625, 0.12841796875, -0.6650390625, 0.005786895751953125, -0.22314453125, 1.02734375, -1.3642578125, 0.64404296875, -0.29833984375, -1.1884765625, -0.0968017578125, -0.5419921875, -0.58447265625, 0.048095703125, 0.395751953125, 0.6259765625, -0.46826171875, 0.402587890625, -1.40234375, 2.091796875, -0.892578125, 0.25634765625, 0.1427001953125, 0.70849609375, 0.51611328125, 0.148681640625, -1.6103515625, -1.8037109375, 0.250244140625, -1.09375, -2.34375, -0.61376953125, -1.8505859375, 1.3896484375, 0.318115234375, -0.0035037994384765625, -1.6513671875, -1.1875, 0.51025390625, 0.418212890625, -0.90869140625, 1.943359375, 0.6171875, 0.2213134765625, 0.5048828125, 0.94873046875, 0.50537109375, -0.14697265625, 1.693359375, -1.3173828125, -0.79931640625, 0.69921875, 1.4814453125, -0.87255859375, -0.396728515625, 0.93017578125, -0.77197265625, 0.050262451171875, 0.2342529296875, 0.339599609375, 0.0732421875, 0.55859375, 0.63916015625, 0.0248260498046875, 0.06915283203125, 0.814453125, 0.1478271484375, -0.2161865234375, 0.95263671875, 0.4638671875, 0.466552734375, 0.282470703125, -0.84765625, 0.76171875, -0.30224609375, -1.5966796875, 0.5888671875, -0.2919921875, 0.07470703125, 2.212890625, -0.2481689453125, 0.8037109375, -1.5576171875, -1.0771484375, 0.751953125, -0.646484375, 0.0347900390625, 0.50244140625, 1.79296875, -0.70068359375, -0.82421875, 1.705078125, -0.426025390625, -0.21533203125, 0.0216522216796875, 1.1357421875, 1.0458984375, -2.236328125, -0.46142578125, 0.2181396484375, -0.54296875, -0.98095703125, 0.4658203125, -0.2406005859375, 0.9521484375, -0.56201171875, 2.28515625, 0.79833984375, 0.57275390625, -0.389892578125, 0.36767578125, -0.583984375, -1.2626953125, 1.44921875, 0.58447265625, 1.330078125, -1.986328125, -0.56396484375, -0.712890625, -0.042205810546875, -1.5166015625, -1.330078125, 1.693359375, 1.0146484375, -0.91015625, -0.300537109375, 0.375244140625, -0.066650390625, 0.032073974609375, 1.275390625, -0.276123046875, 0.06396484375, 0.708984375, 1.3544921875, 0.9580078125, -2.654296875, 2.474609375, -0.8740234375, 0.64013671875, -0.33154296875, -0.546875, -1.18359375, -0.355712890625, -1.6767578125, 0.2286376953125, -0.6904296875, 0.79443359375, 1.2822265625, 0.01141357421875, 0.10272216796875, -0.18896484375, 0.90673828125, -1.9619140625, -0.1680908203125, 0.72021484375, -0.26708984375, -0.77294921875, -0.7412109375, -0.9658203125, 1.091796875, -0.546875, 0.5869140625, 1.185546875, 0.86083984375, -0.95458984375, -1.2880859375, -0.1920166015625, -0.03497314453125, -1.37109375, -1.2822265625, 0.890625, -0.40625, 2.12890625, 1.6328125, 0.39599609375, -1.1669921875, 0.01617431640625, -0.56298828125, -0.373046875, -0.2252197265625, 0.84619140625, -2.0703125, 0.720703125, -0.69580078125, 0.0230560302734375, -1.2998046875, -1.2177734375, 0.45703125, -1.048828125, 0.07000732421875, 0.50244140625, 0.72216796875, -1.4853515625, 1.6689453125, -1.953125, 0.54443359375, 1.3671875, 1.2861328125, -0.347900390625, 0.7607421875, -0.2110595703125, 0.174072265625, 0.6875, 0.25439453125, 0.62744140625, -0.78759765625, 0.85498046875, 0.418701171875, 3.123046875, -0.53173828125, 2.060546875, 0.247802734375, -0.141845703125, -0.939453125, -1.083984375, -1.49609375, -0.4306640625, 0.2281494140625, -1.24609375, 1.1201171875, 0.482177734375, 1.2314453125, -1.1044921875, 1.509765625, 0.3583984375, 1.87890625, -0.461669921875, -1.1591796875, -0.78564453125, -0.1304931640625, -0.31640625, -1.708984375, 0.317626953125, 0.8212890625, -0.171630859375, 0.85888671875, -1.158203125, 0.436279296875, -1.431640625, -0.85205078125, 1.533203125, 0.466796875, -2.115234375, 0.36572265625, 0.485595703125, -0.9697265625, -0.1656494140625, 1.6513671875, 0.45703125, -0.77294921875, -0.7470703125, 0.50830078125, 0.994140625, 0.95556640625, 0.29150390625, 0.8125, 0.14892578125, -0.03466796875, 0.0086822509765625, 1.3193359375, 0.939453125, -1.4111328125, -0.185546875, 0.60009765625, 0.4423828125, -1.310546875, -0.5986328125, -0.107177734375, -0.060821533203125, 0.52734375, -0.440185546875, 0.431884765625, -0.399169921875, -1.693359375, -0.01898193359375, -0.1387939453125, -1.1943359375, -1.548828125, -0.73974609375, -0.70361328125, -0.216552734375, 1.87109375, -1.212890625, 0.65185546875, 0.398193359375, 0.1595458984375, 0.3505859375, 1.3486328125, 0.7529296875, 1.4755859375, 0.325927734375, -0.75927734375, -0.7587890625, 1.5888671875, -0.1337890625, -1.033203125, 0.74658203125, 0.0677490234375, -0.50439453125, 0.2440185546875, 1.6298828125, -0.2242431640625, 0.44970703125, -0.2113037109375, -0.054290771484375, -1.416015625, -1.1826171875, -0.256591796875, -0.078125, -0.2281494140625, 0.61376953125, 0.95263671875, 0.73876953125, -1.71875, 0.05511474609375, -0.50390625, 2.18359375, 2.2578125, -1.263671875, 0.427490234375, -0.11041259765625, -0.49560546875, -0.904296875, -0.4599609375, 0.196533203125, -0.27099609375, -0.18701171875, -0.469482421875, 0.42431640625, -0.166259765625, 0.04632568359375, 1.453125, 0.982421875, -1.005859375, -0.81591796875, 1.4931640625, -1.4921875, 1.2333984375, 1.693359375, 0.339599609375, -1.408203125, 1.068359375, -1.47265625, -0.92333984375, -0.66943359375, 0.55908203125, 2.4765625, -0.81396484375, 0.96484375, 0.414306640625, 0.583984375, -1.1591796875, -0.6142578125, 0.71923828125, 0.1649169921875, 0.58203125, -1.220703125, 0.435791015625, 0.53369140625, 0.66455078125, 0.294921875, 0.004199981689453125, -0.2626953125, -1.748046875, -1.875, 1.9091796875, -2.6171875, -0.53466796875, -0.1845703125, 1.1044921875, -0.9873046875, 1.4677734375, -0.1983642578125, 1.0751953125, 0.76220703125, -1.8720703125, 0.04144287109375, 0.458984375, -1.1064453125, -1.1953125, 0.53564453125, -0.71923828125, -0.61376953125, 0.11859130859375, 2.12109375, -1.1767578125, 0.69775390625, -0.08575439453125, -1.685546875, 1.080078125, 0.744140625, -1.4560546875, -0.113037109375, -0.376708984375, -2.263671875, -0.49951171875, 1.5458984375, 0.240234375, -0.7412109375, -1.17578125, 1.091796875, -1.5849609375, -0.0426025390625, 1.115234375, 1.1904296875, -1.6982421875, -1.029296875, -1.4736328125, 1.505859375, -0.361083984375, 0.0081329345703125, 1.1904296875, -0.94921875, 0.3525390625, 0.62646484375, -0.8564453125, -2.86328125, 0.6123046875, 1.14453125, -0.05853271484375, 0.53515625, 1.751953125, 0.390869140625, -1.1640625, 0.599609375, 0.07904052734375, -1.421875, -0.8291015625, 2.34765625, 0.3916015625, -0.9912109375, 0.66162109375, 0.0145111083984375, 0.10296630859375, 0.486328125, 0.217529296875, -0.6865234375, 0.8447265625, 0.2335205078125, 1.7099609375, -0.26123046875, -1.65234375, -0.51416015625, 1.3408203125, -1.2490234375, 1.0439453125, -0.32763671875, -1.05078125, 0.1473388671875, -0.81689453125, -1.2861328125, -0.61962890625, 0.1060791015625, 0.134521484375, -1.083984375, 0.2484130859375, -0.314208984375, 0.34716796875, 0.1865234375, 0.5078125, 0.41015625, 0.6220703125, 0.51904296875, -0.2010498046875, 0.4697265625, 0.7490234375, -0.43994140625, 0.239501953125, 0.7724609375, 0.53076171875, -0.07208251953125, -0.345703125, 1.23046875, -0.9189453125, -0.271728515625, -0.7353515625, 0.3203125, 0.73095703125, 0.3974609375, 0.82275390625, -1.9345703125, 1.48046875, -0.1728515625, -0.51318359375, -1.09765625, -1.7421875, -1.48046875, 1.7890625, 0.496826171875, 1.8203125, 0.2288818359375, -0.912109375, -0.68408203125, 0.23681640625, 0.8505859375, -1.0341796875, -0.49169921875, -0.045074462890625, 0.243408203125, 0.5556640625, -1.2958984375, 0.5830078125, -0.396240234375, -0.09747314453125, 0.169677734375, -0.73388671875, 1.283203125, 0.591796875, 0.63818359375, 1.9580078125, -0.6298828125, 0.5595703125, 1.77734375, -1.8515625, 0.1492919921875, 0.0276641845703125, -0.71435546875, -0.341552734375, 0.94580078125, -1.4423828125, -0.40771484375, 0.537109375, -0.7255859375, -0.334716796875, -0.469970703125, -0.74853515625, 0.7919921875, -0.11529541015625, -0.52490234375, 0.1680908203125, 0.357177734375, -1.4140625, -0.333740234375, -0.0787353515625, 1.1943359375, 1.005859375, -0.88232421875, -1.9140625, 1.0537109375, 0.64208984375, -0.94287109375, 0.0023441314697265625, -2.26171875, -2.98046875, 0.487548828125, -0.908203125, -0.619140625, 0.330810546875, -1.220703125, -0.9306640625, -0.1033935546875, 0.7919921875, -0.325927734375, 0.04705810546875, 0.6923828125, -0.60009765625, -1.0283203125, 1.3125, -1.5146484375, -0.67236328125, -0.59228515625, 0.806640625, -0.062286376953125, -0.2415771484375, -0.2135009765625, 0.564453125, 2.1796875, -0.47607421875, 0.009185791015625, 0.385009765625, 1.0166015625, -1.2685546875, -1.103515625, -0.54296875, 0.833984375, -1.814453125, 1.208984375, -0.4443359375, 1.033203125, 1.9619140625, 2.11328125, 1.3046875, 1.427734375, 1.560546875, 0.450439453125, -0.38623046875, -0.259765625, -2.55078125, -0.307373046875, 0.1412353515625, 0.3095703125, -0.44482421875, -0.38818359375, 0.869140625, -0.39404296875, 0.363037109375, -1.4091796875, 0.66015625, -0.380859375, -0.68994140625, 0.279296875, -0.70849609375, 0.77197265625, -0.2060546875, -0.130126953125, 1.421875, 0.0225677490234375, -0.87548828125, -0.138671875, -1.91796875, -0.350341796875, -0.6689453125, -0.45751953125, -1.9111328125, 0.428466796875, 0.63671875, 0.37890625, 1.0927734375, 0.513671875, 0.39306640625, -0.78857421875, -0.9462890625, -0.0173492431640625, 0.40576171875, 1.32421875, -0.360595703125, -1.1201171875, -0.75244140625, 0.2353515625, 0.9814453125, -1.3017578125, 0.003551483154296875, 0.0867919921875, 0.4453125, 1.9765625, 0.99951171875, -1.1435546875, -1.103515625, -1.076171875, 0.233154296875, 1.5126953125, -0.5439453125, 0.6357421875, -1.2099609375, 0.5458984375, -1.1435546875, 0.4072265625, -0.83984375, -1.9580078125, 0.020233154296875, 0.53759765625, 1.03125, -0.13134765625, -2.052734375, 1.173828125, 0.64990234375, -0.0791015625, 1.3837890625, 0.141357421875, 0.92724609375, -0.292236328125, -0.9130859375, 0.78857421875, 0.333740234375, -0.398681640625, 0.00724029541015625, -1.28125, -1.35546875, -0.4033203125, 1.1181640625, -0.2412109375, 1.5751953125, 0.56396484375, -1.474609375, -0.74609375, -1.6572265625, 0.89404296875, 0.0994873046875, -0.6220703125, 0.2685546875, 1.5322265625, 1.3984375, -0.03814697265625, 1.21484375, -0.054473876953125, -1.2412109375, -0.2122802734375, 0.255859375, 1.78515625, -0.83056640625, 2.251953125, -0.198486328125, -0.52978515625, 0.1807861328125, -0.393310546875, 0.58349609375, -1.2529296875, 0.73828125, 1.09375, 0.84423828125, 0.305419921875, -0.76025390625, -0.2442626953125, -0.740234375, -1.1318359375, 0.415771484375, 1.0693359375, 0.59130859375, 0.52197265625, 0.11944580078125, 0.265380859375, 0.359375, -0.64111328125, 0.85693359375, -1.1513671875, -1.96875, 0.31982421875, -0.393310546875, -0.243896484375, 0.44580078125, -0.0440673828125, 1.451171875, 0.34375, -0.2344970703125, -0.134033203125, -1.9619140625, 0.32958984375, -0.061767578125, 1.484375, 0.58740234375, -1.3466796875, 0.07879638671875, 0.1558837890625, 0.27490234375, -0.70166015625, 0.005878448486328125, 1.0595703125, 0.06024169921875, -0.33544921875, -0.390625, 1.0419921875, 1.9296875, -1.7294921875, 0.3369140625, 0.98046875, -0.96533203125, 0.9306640625, 0.7900390625, 1.3759765625, 0.27685546875, 0.9375, -0.2587890625, -0.2314453125, -1.4033203125, -0.44873046875, 1.373046875, -0.59033203125, 1.0283203125, -0.6513671875, 0.011810302734375, 0.5673828125, -0.67138671875, 1.744140625, 0.2335205078125, 0.61474609375, -0.70361328125, -0.07049560546875, -0.546875, 2.2109375, 1.185546875, 0.9375, 0.439208984375, 0.9501953125, 0.34326171875, -0.421875, 0.75439453125, 0.36376953125, -1.2783203125, -0.94970703125, -0.0222625732421875, -0.111572265625, 0.1627197265625, 1.9208984375, 1.5341796875, -0.52685546875, -0.6962890625, 0.128662109375, 0.253662109375, -1.177734375, 1.25, -0.44677734375, -0.313232421875, -0.308837890625, 0.058013916015625, -0.302734375, 0.050384521484375, -0.4970703125, 0.60498046875, -0.09014892578125, -0.36279296875, 0.1016845703125, 0.1510009765625, 0.8447265625, 2.0859375, -0.59033203125, 1.603515625, 0.396484375, 1.568359375, -0.59423828125, 0.151123046875, 0.254638671875, -0.1787109375, -1.0126953125, 0.52197265625, -1.49609375, 0.63037109375, 0.136962890625, -0.422119140625, 1.5546875, -0.96435546875, -0.2705078125, 0.3916015625, -0.147216796875, -1.935546875, 0.03851318359375, 0.80517578125, 0.99365234375, 0.30419921875, -0.94873046875, -1.1953125, 0.497314453125, -0.53662109375, -0.4404296875, 0.38232421875, -0.2216796875, -2.76171875, -1.421875, -0.49560546875, -1.171875, 0.05853271484375, -1.498046875, 1.01953125, 0.57666015625, -0.1873779296875, -0.29248046875, -1.6728515625, -2.57421875, 0.9248046875, 0.51416015625, -0.68115234375, -0.62109375, -0.225341796875, -1.5078125, 1.0087890625, -0.91162109375, 0.81640625, 0.11932373046875, -0.484375, 0.845703125, -0.480224609375, -2.205078125, 0.51953125, -1.6318359375, 0.09185791015625, -0.46826171875, -0.446533203125, -1.294921875, 0.0282745361328125, -0.1904296875, 0.1309814453125, -1.2451171875, 0.60400390625, 1.3916015625, 1.779296875, -1.2099609375, -0.380859375, -0.4716796875, 0.250732421875, -0.75390625, 0.0819091796875, -1.2421875, 0.9814453125, 0.139892578125, -2.029296875, 0.1632080078125, -0.0711669921875, 0.96337890625, -1.265625, 0.53466796875, 0.12481689453125, 1.5087890625, -1.1884765625, -1.2900390625, -0.0999755859375, -1.3740234375, -0.373291015625, -0.18701171875, 0.8408203125, -0.463134765625, 1.0244140625, 1.43359375, 0.279052734375, -0.0738525390625, 1.45703125, -1.7060546875, 2.419921875, 1.15234375, 0.99169921875, -0.1259765625, -0.38232421875, -2.388671875, 1.8828125, 0.69140625, 1.1552734375, -0.18212890625, -1.2822265625, 1.2626953125, -0.6943359375, 1.1943359375, -1.4169921875, -0.65966796875, -0.0765380859375, -1.0654296875, 0.02032470703125, 1.1396484375, -2.216796875, 1.0263671875, -1.51953125, -0.1727294921875, -1.7041015625, 1.9736328125, -1.2109375, -0.85791015625, 1.1318359375, 0.80615234375, 0.53271484375, 2.0390625, -0.097900390625, -0.56591796875, -0.51513671875, -0.037628173828125, -0.7568359375, -0.67529296875, -2.255859375, -0.28857421875, 0.2078857421875, 0.227294921875, 1.7080078125, 0.2467041015625, 1.169921875, -0.0076904296875, 0.8349609375, -0.175048828125, -1.1201171875, -0.3623046875, -0.0899658203125, 0.77880859375, 1.1015625, 1.318359375, -1.0888671875, 0.7998046875, 0.56396484375, 0.697265625, -0.6357421875, -0.347412109375, 1.2431640625, 0.87451171875, -0.25390625, -0.254638671875, 0.74560546875, -0.92919921875, -2.044921875, -0.5732421875, 0.61376953125, -2.6640625, -0.0694580078125, -1.3974609375, 1.0791015625, -0.398193359375, 0.69482421875, -0.58935546875, -0.374755859375, 0.32666015625, -0.77783203125, 1.623046875, 2.087890625, -0.127197265625, -0.6494140625, 1.478515625, -0.064208984375, 1.046875, -1.310546875, -1.4150390625, -1.1025390625, -0.37353515625, -1.3544921875, -0.53955078125, -2.37109375, -0.03680419921875, 1.4169921875, -1.5380859375, -0.22802734375, -1.158203125, 0.35205078125, -0.3486328125, 1.31640625, 0.156494140625, 0.9443359375, 1.2978515625, 0.412353515625, 1.5625, 0.826171875, 0.8818359375, 0.90966796875, -0.1536865234375, 0.76708984375, 1.9619140625, -0.061553955078125, 1.87890625, 0.60400390625, -2.236328125, -1.25390625, 2.384765625, -0.41455078125, -0.3564453125, 1.7626953125, -0.410888671875, -0.079833984375, 0.392578125, 0.2998046875, 1.3896484375, 1.4521484375, -0.037261962890625, 0.689453125, -0.32568359375, -2.01171875, -0.0119476318359375, -0.09466552734375, 0.1943359375, -0.38671875, 0.057952880859375, 1.15234375, 0.5986328125, 0.78662109375, -0.25390625, 0.08929443359375, -0.8232421875, 0.08905029296875, 0.0941162109375, 2.18359375, 1.37890625, -0.20263671875, 1.0234375, 0.578125, -0.6767578125, -0.63134765625, 0.281494140625, -0.59326171875, 0.4501953125, 1.0283203125, 0.317626953125, -1.9921875, -0.87255859375, -1.8408203125, -0.27490234375, 0.43212890625, 0.1004638671875, 0.6123046875, 1.0849609375, -0.259521484375, 1.3232421875, 1.4150390625, 0.1839599609375, -0.299072265625, 0.87060546875, 1.453125, -1.8740234375, -1.7734375, -0.99951171875, 0.0032062530517578125, -0.74658203125, -1.8984375, -0.409423828125, -0.319580078125, 0.50732421875, -0.1700439453125, -0.3505859375, 0.85009765625, -0.467529296875, -2.12890625, 0.431640625, 1.5927734375, -0.67919921875, -2.21484375, -1.61328125, -0.7421875, -0.07049560546875, 0.15380859375, -0.333984375, 1.4208984375, 1.78125, -1.4765625, 0.6416015625, 0.345458984375, -0.21142578125, -0.11785888671875, -1.00390625, -0.7392578125, 1.1142578125, 1.1552734375, -0.71923828125, 0.24169921875, -1.98046875, 1.611328125, 1.544921875, -1.19140625, 0.7587890625, 0.483642578125, -0.4384765625, 1.4990234375, -1.564453125, -0.80615234375, -1.400390625, 1.1240234375, -0.65380859375, -0.54833984375, -0.66796875, 0.9326171875, -0.45166015625, 0.9521484375, -0.09466552734375, -0.0107574462890625, 0.2030029296875, -0.433837890625, 1.15234375, -0.0159454345703125, 1.0986328125, 0.1610107421875, 0.126953125, -0.60693359375, 0.27294921875, -0.84619140625, 0.283203125, 0.436279296875, -0.6884765625, -1.0009765625, -1.0302734375, -0.83837890625, -2.0546875, 0.402587890625, -1.41015625, 0.464111328125, 0.08880615234375, 1.259765625, -0.7041015625, -1.6259765625, -0.94580078125, 0.33154296875, 2.2109375, 1.279296875, -0.06024169921875, -0.378173828125, 0.3623046875, 1.392578125, -1.1484375, -1.9755859375, -0.05120849609375, 0.275634765625, 0.6630859375, 0.3095703125, 1.357421875, 3.03515625, -0.80810546875, 1.2607421875, -0.300537109375, -1.51171875, -2.0390625, 0.205810546875, -0.325927734375, -0.8583984375, -1.2919921875, 0.445556640625, 0.273193359375, 0.7392578125, 0.85400390625, 0.283447265625, -0.4638671875, 1.34375, -1.5224609375, -0.093994140625, 0.6142578125, 1.30859375, 0.8232421875, 0.767578125, 1.3818359375, -0.215576171875, 0.296630859375, 1.5517578125, 0.583984375, 2.00390625, 0.5078125, -0.1085205078125, -0.84716796875, 0.00867462158203125, -0.90673828125, 0.615234375, 1.6552734375, -0.7021484375, 0.560546875, 0.176025390625, -1.015625, 0.1468505859375, -1.037109375, 2.802734375, -1.0615234375, -1.5234375, -0.9150390625, -0.9765625, -1.40234375, -0.203857421875, 0.040557861328125, 1.53515625, -0.006195068359375, 0.003452301025390625, 0.88818359375, 1.85546875, -0.82177734375, 0.1162109375, 1.2177734375, -0.1927490234375, -0.796875, -1.7734375, 0.5439453125, -0.76171875, -0.78076171875, -2.564453125, 0.11639404296875, 0.765625, -0.14453125, 0.62060546875, -0.348876953125, -0.156982421875, 3.46875, 1.8388671875, -1.5068359375, -1.28125, 0.98388671875, -0.190673828125, 1.275390625, -0.74169921875, -0.4697265625, 1.375, 0.9111328125, -0.2396240234375, -0.399658203125, 0.83349609375, 1.1611328125, -0.265625, 0.3935546875, -2.048828125, 0.2939453125, 0.2509765625, 1.587890625, 0.79052734375, 0.12127685546875, -0.6572265625, -0.7080078125, 1.2060546875, 0.70361328125, 1.5166015625, -0.3896484375, 0.42626953125, 0.380126953125, 1.220703125, 0.266357421875, -2.56640625, 0.74609375, -0.391845703125, 0.69384765625, -0.2425537109375, -0.50341796875, 1.15625, -1.3232421875, 0.71630859375, 0.0736083984375, 0.26220703125, -1.056640625, -0.86376953125, -0.970703125, -0.9521484375, 2.240234375, -0.147216796875, 0.7373046875, 0.501953125, 1.8369140625, 2.494140625, 0.5859375, 0.434326171875, 0.2401123046875, -0.083740234375, -0.74169921875, 0.49609375, 1.2958984375, -1.310546875, 0.2890625, -0.229736328125, -0.275146484375, -0.2203369140625, -0.8173828125, 1.337890625, 0.0279388427734375, 0.94482421875, 1.5458984375, 0.4453125, -0.1395263671875, -0.55908203125, 0.33837890625, -0.986328125, 1.2998046875, -0.12109375, 0.5791015625, -0.7802734375, -0.19580078125, 0.87646484375, -1.6337890625, -0.176513671875, 0.91650390625, 0.267578125, -1.046875, 0.110595703125, 0.8203125, 1.234375, -0.68359375, -0.334716796875, -0.410888671875, 0.50732421875, -0.324462890625, -0.5595703125, 1.099609375, 0.64453125, 0.8583984375, 0.8349609375, -0.5185546875, 0.5732421875, -1.080078125, 0.2181396484375, -1.431640625, 0.192138671875, -1.2060546875, 0.427978515625, -1.7666015625, -0.93017578125, -1.1708984375, -0.69384765625, -1.353515625, 1.1171875, 0.52392578125, -0.40478515625, 1.0673828125, -1.57421875, 0.213134765625, -0.81201171875, 0.41357421875, 0.55712890625, 0.79541015625, -0.42041015625, -0.469970703125, -0.865234375, -0.98388671875, 1.353515625, -1.2705078125, -1.1142578125, 0.91748046875, -0.2373046875, -2.43359375, 0.66064453125, -0.1331787109375, -0.771484375, 0.556640625, -1.0771484375, 0.658203125, -1.5107421875, -0.52685546875, 1.0927734375, -0.8408203125, -0.0572509765625, -0.54052734375, -0.908203125, 1.1630859375, 0.26611328125, 1.2744140625, 1.5556640625, 2.330078125, -0.1778564453125, 0.031829833984375, -0.050872802734375, -1.1181640625, 0.1934814453125, 0.38330078125, 0.186279296875, -0.1634521484375, 2.080078125, -1.58203125, -1.025390625, 1.03125, 0.1597900390625, -0.68603515625, -0.2156982421875, -0.2423095703125, 1.50390625, 0.447021484375, -0.53662109375, -0.182373046875, -2.091796875, 1.337890625, -0.75146484375, -0.580078125, 1.5419921875, -1.6240234375, 0.3193359375, 1.1748046875, 0.10693359375, 0.89697265625, -0.57470703125, 1.5537109375, 0.467529296875, 1.8466796875, 0.1878662109375, -0.1575927734375, -0.46826171875, 1.0341796875, -0.10565185546875, -0.56689453125, 1.142578125, -0.2061767578125, 2.53125, 0.25830078125, 0.6640625, 0.66650390625, -0.1876220703125, 1.0576171875, 0.74072265625, -1.76953125, 2.025390625, 0.1884765625, 0.04388427734375, -0.32080078125, -0.338623046875, -1.0693359375, -0.0950927734375, -0.01190948486328125, 1.6494140625, 0.1822509765625, -2.17578125, 0.80078125, -0.439208984375, -1.158203125, -1.775390625, -0.304931640625, -0.5947265625, -0.3779296875, 0.2213134765625, -0.94970703125, -0.456298828125, 0.047637939453125, 0.473876953125, 0.402099609375, 0.6884765625, -0.62255859375, 0.011566162109375, 0.187744140625, 0.57958984375, -0.08099365234375, 0.0311431884765625, -1.2822265625, -0.8916015625, -1.4853515625, -0.97509765625, 0.65625, -0.0662841796875, -0.354248046875, -0.95654296875, -1.0126953125, -2.0859375, 0.143310546875, 0.2064208984375, -0.857421875, 0.2122802734375, -0.042572021484375, -1.2421875, -2.9375, 0.91943359375, -1.0888671875, -1.4365234375, -0.48291015625, -0.5732421875, -1.1708984375, 0.01317596435546875, 0.6884765625, -1.1435546875, -0.8701171875, 0.1004638671875, 1.43359375, -0.0751953125, 0.179443359375, 0.350830078125, 0.1712646484375, 0.91943359375, -0.337158203125, -2.052734375, -0.1318359375, -1.0146484375, 0.8447265625, -0.18017578125, 0.21728515625, -0.89794921875, -0.57568359375, 0.556640625, 1.568359375, -1.0625, 1.7099609375, -0.1884765625, 0.650390625, -0.99267578125, 1.201171875, -0.378173828125, -1.2080078125, 0.92041015625, 0.6826171875, -1.4990234375, 0.65966796875, -0.11737060546875, 0.875, 1.2998046875, -1.4619140625, 0.12042236328125, -0.11956787109375, 0.09210205078125, 0.474609375, 1.033203125, -0.0265045166015625, -1.927734375, 0.75732421875, 0.9326171875, 0.0244140625, 0.6123046875, -0.138671875, 0.262451171875, 0.666015625, 0.16650390625, -2.509765625, 0.62890625, 0.28369140625, -0.60888671875, -0.387451171875, -0.060089111328125, 0.6474609375, -0.273681640625, -2.146484375, -0.50634765625, 1.14453125, -0.252685546875, 0.0694580078125, 1.1474609375, -0.13037109375, 1.4462890625, -0.64501953125, -0.5185546875, -0.6240234375, 1.7919921875, -0.7099609375, 1.349609375, -0.4970703125, 0.99267578125, 1.4453125, -0.1591796875, 0.25341796875, 1.2353515625, -1.35546875, -0.92529296875, -0.462158203125, 0.85791015625, 1.6064453125, 0.595703125, 0.68701171875, 0.6474609375, -1.13671875, 0.75390625, -0.1324462890625, -1.126953125, 1.55859375, -1.6875, -0.101318359375, -2.25, 0.4306640625, -0.82666015625, 0.479248046875, 2.296875, 0.4658203125, -0.53857421875, -0.70849609375, 0.5947265625, 0.0484619140625, 0.0217132568359375, -2.228515625, -0.1668701171875, 1.404296875, 0.348388671875, -0.6630859375, 0.71875, 0.53515625, 1.1630859375, 0.59326171875, 0.061767578125, 1.8193359375, -0.045562744140625, -0.9208984375, -0.307373046875, 0.212890625, -1.3134765625, 0.57861328125, -0.83251953125, -0.1435546875, -0.93359375, -0.5849609375, -1.5400390625, -1.7685546875, 0.252685546875, -0.890625, -0.08447265625, -0.8681640625, -0.548828125, -1.2392578125, -0.1177978515625, 0.8671875, 1.6787109375, 0.51611328125, -0.6396484375, -1.626953125, -0.320068359375, -0.85009765625, 0.8994140625, 0.99658203125, 0.4619140625, 1.7197265625, -0.275634765625, 1.6181640625, 0.61572265625, 1.1611328125, 1.259765625, -0.87744140625, 0.76318359375, -0.4150390625, 0.6865234375, -2.31640625, -2.3671875, 0.22607421875, -0.359619140625, -0.82275390625, 1.16015625, 0.30810546875, 0.260009765625, 0.37890625, 0.1551513671875, 0.7412109375, -0.027923583984375, 1.1240234375, -1.2421875, 1.3994140625, -0.64111328125, 0.53564453125, 0.8017578125, -0.8349609375, 1.330078125, 1.1337890625, 0.6611328125, 0.5244140625, -0.876953125, 0.3154296875, -0.1075439453125, -0.11517333984375, 0.76806640625, -1.2275390625, 0.2447509765625, -0.0726318359375, 1.333984375, 0.058013916015625, -1.3564453125, -1.1181640625, 2.041015625, -1.4658203125, 0.96875, 0.0182037353515625, -1.9052734375, -1.236328125, -1.021484375, -0.87744140625, -0.14892578125, -0.298583984375, -0.394775390625, -0.9248046875, -0.1229248046875, 0.48291015625, -0.705078125, 0.88427734375, -1.3798828125, 0.455322265625, 0.33642578125, -0.05963134765625, -0.81982421875, -0.9521484375, 1.6875, -0.335693359375, 1.505859375, -1.0546875, -0.1083984375, 0.38037109375, -0.978515625, 0.94384765625, -0.8818359375, 0.02239990234375, 0.1322021484375, 0.343017578125, -0.7724609375, -0.72900390625, -1.6865234375, -1.0419921875, -0.56201171875, 0.86767578125, 0.430908203125, -0.64453125, 0.951171875, 0.2425537109375, 1.4599609375, -0.62158203125, -0.552734375, 1.6552734375, -0.9814453125, -0.9873046875, -0.94873046875, -1.701171875, 0.3212890625, -0.5771484375, -1.130859375, -0.59765625, 0.08984375, 1.0927734375, 0.68505859375, -1.5791015625, -1.75, 1.2431640625, -0.95556640625, -0.1944580078125, -0.255126953125, 1.8154296875, -0.58984375, -2.08203125, -0.390625, 0.490966796875, 0.5283203125, -0.255126953125, -1.0205078125, -0.57763671875, -0.72412109375, -0.64990234375, 0.00730133056640625, -0.8896484375, -0.4794921875, 0.1429443359375, 0.57568359375, -0.201904296875, 0.1483154296875, -0.150390625, -0.282470703125, 0.2042236328125, -0.242919921875, 0.7734375, -0.055816650390625, -1.52734375, -1.6044921875, -1.1982421875, -0.74755859375, 0.104736328125, -1.5673828125, -2.119140625, -0.318115234375, 1.5244140625, 1.4365234375, -0.4296875, 0.86083984375, 0.646484375, -0.5654296875, 1.0751953125, -0.2476806640625, 0.59716796875, 0.059844970703125, -2.09375, 0.1905517578125, -0.62353515625, 2.224609375, -0.63623046875, 0.1531982421875, 0.7734375, -0.84912109375, 1.1181640625, 0.74658203125, -0.52490234375, -1.0185546875, -1.0263671875, 0.484619140625, 1.35546875, -0.5146484375, -0.07135009765625, -0.054595947265625, 1.2958984375, -0.317626953125, -0.43896484375, -0.0003902912139892578, -0.0003724098205566406, -0.60205078125, -1.4931640625, 1.4443359375, -0.56689453125, -0.050018310546875, 0.75146484375, 0.62158203125, 0.7490234375, -0.393798828125, 1.681640625, -1.1064453125, -0.69580078125, 0.994140625, -0.06884765625, -0.5009765625, 0.642578125, 0.005123138427734375, 0.7802734375, -1.1875, 1.126953125, -1.0654296875, -0.00725555419921875, -0.99267578125, -0.603515625, 1.5078125, -0.5927734375, 0.8740234375, 0.99169921875, -0.277587890625, 1.15625, 1.0068359375, -0.810546875, 1.3515625, -1.03125, 0.207763671875, -1.41015625, -0.3466796875, -0.42041015625, 0.057647705078125, -0.61328125, 1.49609375, 0.392333984375, -0.018310546875, 0.78173828125, 1.384765625, -0.39111328125, 1.68359375, -0.4375, 0.49658203125, 1.32421875, -0.11383056640625, -2.08984375, -1.2138671875, -1.8662109375, 0.042266845703125, 0.169189453125, -0.05670166015625, -1.984375, -1.4033203125, 0.405517578125, -0.806640625, -0.09735107421875, -2.20703125, -1.57421875, 0.1282958984375, 2.578125, 1.3134765625, 1.1591796875, -0.26953125, 1.66796875, 2.130859375, -0.1287841796875, 0.002410888671875, 1.318359375, 0.78466796875, 0.289306640625, 0.54736328125, 0.425048828125, -0.70556640625, 2.0390625, 0.60107421875, 0.36376953125, -1.4306640625, 0.1627197265625, -0.220458984375, 0.395263671875, 0.1268310546875, -0.541015625, 0.67822265625, -0.9296875, -0.74755859375, 0.86376953125, 1.88671875, -0.5390625, 0.39501953125, 0.501953125, -1.3046875, -0.2486572265625, -0.65185546875, 0.335693359375, -0.302734375, -0.71142578125, -0.71142578125, -0.791015625, 0.78662109375, 0.580078125, -0.007717132568359375, -0.560546875, -0.31689453125, 0.34521484375, -0.6640625, -1.65625, -1.6376953125, -0.89892578125, 0.294189453125, 0.7177734375, 0.378173828125, -2.01171875, 1.6064453125, -1.4580078125, -0.80859375, -0.6025390625, 1.17578125, 0.323974609375, -1.298828125, -1.55859375, 0.40283203125, 0.312744140625, -1.455078125, 0.8681640625, 1.140625, 0.413818359375, 1.5576171875, -1.1904296875, 0.47412109375, 2.09765625, 0.71630859375, -0.91943359375, -0.9443359375, 1.318359375, 1.5048828125, -0.58251953125, -0.6201171875, 1.0087890625, 0.77880859375, -0.132080078125, -0.37744140625, -1.064453125, -1.36328125, -0.09307861328125, 0.98779296875, -1.78515625, -0.7734375, -0.26318359375, -1.1455078125, -1.30078125, -0.5595703125, -0.5966796875, 0.06048583984375, 0.44921875, -1.93359375, -0.84716796875, 0.96728515625, 0.25341796875, -0.53076171875, 0.0310211181640625, -1.4033203125, -0.2188720703125, 0.24072265625, -0.99560546875, 1.0087890625, 0.61767578125, -0.69189453125, -0.25830078125, 0.5751953125, -0.397705078125, 0.66650390625, 0.73828125, 1.28515625, 1.0185546875, -0.7802734375, 0.42626953125, 0.297607421875, -2.09765625, -0.91259765625, -0.71630859375, -2.263671875, 0.94921875, 0.5263671875, 0.3984375, -0.006923675537109375, 0.08721923828125, -0.72412109375, -0.1146240234375, 0.37744140625, -0.5673828125, 1.4873046875, 0.050628662109375, 1.1494140625, 1.5673828125, 0.93701171875, 0.4501953125, -0.19189453125, 0.1328125, -0.72216796875, 0.4580078125, 0.6669921875, -0.54443359375, -1.05859375, -1.099609375, 0.1297607421875, 1.1181640625, -0.1451416015625, -1.2333984375, -0.22900390625, 0.2210693359375, -0.73681640625, 0.229248046875, -0.41845703125, 0.7021484375, -0.2418212890625, 1.0234375, -0.1436767578125, 1.3056640625, -0.62548828125, -0.35791015625, -1.451171875, -0.361328125, -0.10565185546875, -0.263427734375, -0.187255859375, -0.64306640625, 0.05029296875, -0.90478515625, -0.11114501953125, 1.263671875, -0.042266845703125, 0.3095703125, 0.07928466796875, 0.0105438232421875, 0.053741455078125, 0.018890380859375, 0.98388671875, -0.42333984375, -0.344482421875, 0.650390625, -1.79296875, -0.0169525146484375, 1.041015625, 0.364501953125, -0.779296875, 0.74365234375, 0.2318115234375, 3.5859375, 0.428955078125, 0.5283203125, 1.3056640625, -1.859375, 1.240234375, -0.08123779296875, -1.0302734375, 1.1845703125, 1.458984375, -1.3203125, -0.544921875, 1.1162109375, 1.1279296875, -0.456298828125, 0.375732421875, 1.2080078125, 0.1630859375, -1.990234375, 0.367919921875, 1.3359375, -0.6484375, 1.1337890625, 1.2060546875, 1.4208984375, 0.174560546875, 0.27099609375, -0.27294921875, -0.038299560546875, -0.08599853515625, -0.1251220703125, -0.441650390625, -0.1361083984375, -0.6318359375, 1.185546875, -0.11883544921875, 0.6376953125, -0.9111328125, 0.12176513671875, 0.685546875, -0.480224609375, -0.1783447265625, 0.58837890625, -0.9375, -0.189453125, 1.162109375, -0.53076171875, 2.794921875, 0.58154296875, -0.5849609375, -0.57275390625, 1.5126953125, 0.1285400390625, 1.087890625, -0.1922607421875, -0.377685546875, 0.50439453125, 0.92333984375, -0.05517578125, 0.78076171875, -0.327880859375, -3.572265625, -0.57861328125, 1.017578125, -0.650390625, 1.0986328125, 0.83544921875, 0.51416015625, 0.5595703125, -0.348876953125, 0.01425933837890625, -1.12109375, 1.435546875, -1.9365234375, 0.845703125, -0.67529296875, 0.463623046875, 0.08062744140625, 0.94384765625, -2.236328125, 2.609375, -0.03228759765625, 0.7490234375, 0.892578125, 2.048828125, 0.84814453125, 0.349365234375, -1.9443359375, 0.76806640625, -0.4267578125, 0.23193359375, 0.52783203125, -2.1328125, -0.568359375, -0.634765625, 1.08984375, 0.06524658203125, -0.18994140625, -0.990234375, -0.73388671875, -0.006298065185546875, -2.140625, 0.7353515625, -1.314453125, 1.3330078125, 1.138671875, -1.44140625, 0.491943359375, 0.181396484375, -0.1767578125, 0.2222900390625, 0.7666015625, -0.99072265625, -1.919921875, 0.79345703125, -0.564453125, -0.82958984375, 2.134765625, -0.94775390625, -1.12890625, 0.880859375, -0.1934814453125, 0.54052734375, -1.2666015625, -0.07940673828125, 0.494140625, 0.5810546875, -1.130859375, 0.2127685546875, -0.93359375, 0.57763671875, -2.78515625, 1.3505859375, -1.251953125, -1.3095703125, 1.8798828125, 2.203125, -0.96337890625, -1.6123046875, -1.5400390625, -1.138671875, 0.373291015625, 0.7802734375, -0.6552734375, 0.27197265625, -0.460205078125, 2.0078125, 0.206298828125, 0.73388671875, -0.71337890625, -0.11419677734375, 1.7216796875, 0.319091796875, -0.38916015625, -0.85693359375, -2.056640625, 0.412109375, 0.1368408203125, -0.44970703125, 0.84375, 0.1541748046875, -1.7451171875, 0.8583984375, -0.229248046875, 0.385986328125, 0.2587890625, 0.11968994140625, -0.478271484375, 0.35400390625, -1.66015625, 0.07928466796875, 1.423828125, 1.6015625, -1.4775390625, -1.3544921875, 0.66162109375, -1.8837890625, 1.2177734375, -1.3291015625, 0.72216796875, 0.426025390625, 0.53564453125, 0.36572265625, 2.119140625, 0.5263671875, -1.4462890625, -0.2030029296875, 1.9326171875, 0.54931640625, -0.53759765625, -1.1220703125, 1.1572265625, 0.135498046875, 1.974609375, -1.138671875, 0.73583984375, -0.28369140625, 0.381103515625, -0.67529296875, -0.48046875, -0.4375, 1.3193359375, 0.53466796875, 0.442138671875, -0.0103759765625, -1.265625, 0.55126953125, 1.416015625, -1.0029296875, -0.3720703125, -0.47265625, 0.1378173828125, 0.88720703125, -0.26708984375, -0.8359375, -0.5830078125, 0.052490234375, -0.5595703125, -0.056549072265625, 0.19775390625, -1.4111328125, -0.398193359375, -1.2080078125, -1.65234375, -0.81103515625, 0.379150390625, -0.156982421875, 1.8525390625, -0.91552734375, -1.748046875, -1.576171875, -0.1610107421875, -2.07421875, 1.7744140625, 0.009521484375, -0.34912109375, -1.923828125, -0.55908203125, -0.0286102294921875, -0.6748046875, -0.783203125, -0.202880859375, -0.401611328125, 1.0966796875, 1.501953125, 0.50439453125, 0.40234375, -1.189453125, 0.99365234375, -0.097412109375, 1.234375, 0.19970703125, 0.3701171875, 0.1617431640625, -0.4765625, 0.767578125, 1.5791015625, -0.34228515625, -0.8662109375, -0.8955078125, -1.0, -0.0311431884765625, 1.5419921875, 0.5830078125, 0.830078125, 0.2724609375, 0.0032558441162109375, 0.640625, -0.345703125, 1.859375, -0.387939453125, -0.2978515625, 0.65576171875, 0.70068359375, 1.6494140625, -1.0263671875, 0.68798828125, -0.60986328125, -0.2413330078125, 0.72314453125, 1.3779296875, 1.1435546875, -0.56103515625, 1.7939453125, 1.091796875, 0.47412109375, -1.0546875, 1.5498046875, -0.01160430908203125, -0.1378173828125, 0.8564453125, 1.6572265625, 0.34619140625, -0.7998046875, -0.0377197265625, -0.20068359375, 0.7470703125, -0.1829833984375, -1.3427734375, -1.0390625, 0.10516357421875, -0.71630859375, 0.36279296875, -0.79052734375, -0.53466796875, 0.5341796875, 0.2744140625, -1.6826171875, 0.259521484375, -1.6708984375, 0.11053466796875, 0.927734375, 0.60888671875, 0.367431640625, -0.0312347412109375, -1.6416015625, 0.362060546875, 0.68701171875, -1.765625, 0.33984375, 0.349853515625, -0.355712890625, 1.775390625, 0.8125, -0.58544921875, -1.6240234375, 1.330078125, 0.8330078125, 0.87646484375, 0.8837890625, -0.205810546875, -0.497314453125, -1.05859375, 1.0263671875, 1.330078125, -2.5234375, -0.61669921875, -1.78515625, -1.1064453125, -1.4189453125, 0.1795654296875, 1.6015625, -2.181640625, 0.40283203125, 0.7607421875, -1.4736328125, -1.17578125, 1.048828125, 0.1314697265625, 0.1904296875, 0.399658203125, 1.80078125, -0.3046875, -1.08203125, 1.583984375, 0.50390625, -0.046051025390625, -0.20849609375, 0.492919921875, -0.2958984375, 1.27734375, 1.0146484375, 0.441650390625, 0.91748046875, -0.7763671875, -0.72607421875, 1.865234375, 0.281494140625, 0.8583984375, 0.71533203125, 1.8232421875, 0.033599853515625, -0.288818359375, -1.08984375, 2.1484375, -0.263916015625, 0.9521484375, 0.09844970703125, 1.626953125, 0.5751953125, 1.5810546875, -0.11090087890625, -1.1552734375, -1.4931640625, -0.2724609375, -0.818359375, 1.1181640625, -0.380615234375, 1.1083984375, 0.6357421875, 0.52490234375, 0.8212890625, 0.80322265625, 1.5009765625, -0.45703125, -2.013671875, -0.128173828125, -0.5234375, -0.60009765625, -0.2705078125, 1.3759765625, -0.470947265625, -0.7119140625, -1.9521484375, -0.1729736328125, -0.8916015625, -1.705078125, 0.5224609375, 1.283203125, 0.6220703125, -0.267822265625, -0.12445068359375, 0.2315673828125, 1.6494140625, -1.41796875, -0.6767578125, -0.3935546875, 0.192626953125, -2.302734375, -0.669921875, -1.951171875, -0.2342529296875, 0.7353515625, 0.04583740234375, 0.327392578125, 2.556640625, -0.86669921875, 0.386962890625, -0.74853515625, 0.2169189453125, -0.34228515625, -0.2330322265625, 0.87646484375, -0.0251617431640625, -0.9423828125, -0.9814453125, 1.443359375, 0.736328125, -1.55078125, 1.30078125, 0.51708984375, -0.06207275390625, 0.71533203125, 0.235107421875, -0.8017578125, 0.58642578125, 1.0390625, 1.822265625, -1.63671875, -0.91650390625, -2.826171875, 0.58740234375, -1.43359375, 0.1494140625, -0.740234375, -1.234375, -0.9638671875, -0.7861328125, -0.2724609375, 0.8564453125, -1.2685546875, 0.77001953125, -0.12469482421875, 0.7900390625, -0.45849609375, 0.096923828125, 0.927734375, -0.412109375, -0.2083740234375, 0.5498046875, -0.327880859375, -0.23779296875, 1.310546875, 0.5908203125, -0.251220703125, 0.82421875, -0.37255859375, -1.130859375, -0.2113037109375, -0.50390625, 2.1015625, -0.42529296875, 1.138671875, -1.529296875, -1.5986328125, 1.412109375, 0.2130126953125, 0.17822265625, 0.0701904296875, -1.2666015625, -2.462890625, 0.255615234375, 1.29296875, -1.0419921875, 1.6962890625, 0.5576171875, 0.052642822265625, 0.630859375, -0.896484375, -1.0771484375, -1.458984375, 2.162109375, 2.22265625, -0.6748046875, 0.1064453125, -1.0400390625, 0.94140625, 0.0006694793701171875, -0.833984375, -0.1822509765625, -1.1416015625, -0.1925048828125, -0.95361328125, -0.53466796875, 0.27783203125, 1.4267578125, 2.03125, -1.259765625, 0.01386260986328125, -0.035552978515625, 0.01560211181640625, 1.951171875, -1.6728515625, -0.5263671875, -0.8154296875, 0.2200927734375, 0.1785888671875, -0.400634765625, 0.984375, -2.1015625, 1.12890625, 0.5419921875, -0.93798828125, 0.7314453125, -0.10516357421875, 0.783203125, 0.134765625, -0.69580078125, 0.75146484375, -0.568359375, -1.8349609375, -0.068115234375, 0.136474609375, 1.1748046875, 0.41943359375, -0.6611328125, -0.58642578125, 0.1268310546875, 0.42822265625, 0.375, -1.361328125, 1.3642578125, -0.61474609375, 2.490234375, -0.89306640625, -1.65625, 1.2255859375, 0.71533203125, 0.61279296875, 0.46484375, 1.8447265625, 1.1953125, -1.3935546875, 2.72265625, 1.03125, 0.751953125, -0.73193359375, -0.51806640625, -0.93212890625, -0.54541015625, -0.472412109375, 1.16015625, 1.8125, -0.30224609375, 0.192138671875, 0.845703125, 1.24609375, -0.46533203125, -1.0732421875, 0.293701171875, -0.1009521484375, -0.09222412109375, 1.138671875, 1.5244140625, -1.12890625, 1.9453125, 0.8271484375, -0.56689453125, -0.76416015625, -0.697265625, 0.9013671875, -0.031036376953125, 0.5263671875, 1.2314453125, 0.6513671875, -0.0958251953125, 0.7822265625, -0.08050537109375, -0.2269287109375, -0.0269927978515625, 0.8857421875, -0.4580078125, -2.013671875, 0.35107421875, -1.015625, -0.94482421875, -0.06103515625, -0.7001953125, -0.505859375, -0.54296875, 0.89208984375, 0.472412109375, -0.55126953125, -0.75, -1.2373046875, -1.095703125, 1.171875, 0.599609375, 1.92578125, 0.9638671875, -0.457275390625, -0.489013671875, -0.158203125, -2.013671875, -0.80810546875, -0.279052734375, -1.267578125, -1.4677734375, 1.1875, -0.453369140625, 0.190673828125, -1.58984375, -0.12164306640625, -0.110107421875, 1.0537109375, -0.1986083984375, 1.130859375, 1.224609375, 1.046875, 0.11346435546875, -0.12396240234375, 0.34521484375, -0.09564208984375, -0.12890625, -0.4208984375, -0.89892578125, 1.76171875, 1.44140625, 1.00390625, -1.203125, 0.3310546875, 0.06414794921875, -0.57177734375, 0.7041015625, 0.71044921875, -0.7587890625, -1.67578125, -1.0859375, -2.267578125, -1.1943359375, 0.943359375, -0.42822265625, 1.0185546875, 1.5927734375, -0.449951171875, -0.39501953125, -0.1815185546875, -0.33154296875, 0.323974609375, -1.07421875, -1.20703125, -1.228515625, -0.0780029296875, 1.0830078125, 1.755859375, -1.37890625, 1.4541015625, -1.578125, 0.2291259765625, -0.90625, -0.310546875, -0.5546875, 0.58203125, -0.61572265625, 0.74169921875, -1.8720703125, 0.595703125, -0.5322265625, 0.48486328125, -0.41845703125, 0.7080078125, -1.6142578125, 1.330078125, -0.9697265625, 0.58203125, -2.015625, 0.173095703125, 0.646484375, 0.3466796875, 0.64599609375, 0.33447265625, -0.935546875, 0.70458984375, -1.33984375, -0.2012939453125, -0.228515625, 0.378662109375, 0.0701904296875, 0.93603515625, -0.497802734375, -1.23828125, 1.2021484375, -0.036712646484375, 0.55517578125, 1.212890625, 0.379638671875, -0.2490234375, -0.62109375, 0.330810546875, -0.391845703125, -0.16650390625, 0.3388671875, -1.11328125, -0.1285400390625, 1.5537109375, 0.039764404296875, 0.4140625, -1.2578125, 1.41796875, -0.2763671875, -0.071044921875, -0.2161865234375, -0.1856689453125, -0.86181640625, 1.1318359375, -0.69482421875, 2.83203125, 1.0439453125, 1.2490234375, 0.011810302734375, 0.4345703125, 1.6923828125, 0.53515625, 0.1343994140625, -1.671875, 1.955078125, -0.9609375, -2.3203125, 1.517578125, 0.78076171875, -0.282470703125, 0.818359375, 1.08203125, 0.80908203125, 0.051300048828125, -0.419921875, 0.50732421875, -0.81982421875, -0.8095703125, 1.40625, -0.06585693359375, 0.68359375, 1.6435546875, 1.09375, 0.5244140625, -0.394775390625, 2.185546875, -0.47900390625, -1.5361328125, 0.366455078125, -0.50732421875, 2.001953125, -0.65966796875, -0.4404296875, -1.1396484375, -2.21484375, -1.14453125, 0.0028533935546875, -0.96240234375, 0.38916015625, 0.3076171875, 0.1351318359375, 1.8759765625, -0.689453125, -0.51904296875, -1.3564453125, 0.287841796875, -0.0361328125, -0.53564453125, -0.296875, -0.22021484375, 0.431396484375, -1.595703125, -0.5263671875, 0.78466796875, -1.615234375, -1.1298828125, -1.0478515625, 0.79248046875, -1.11328125, 0.5537109375, 0.134521484375, 1.5888671875, -2.208984375, -0.08892822265625, 1.7587890625, 1.2802734375, 0.464111328125, 0.0966796875, 0.771484375, -0.42822265625, 2.0078125, -0.1944580078125, -0.0036067962646484375, 0.1639404296875, 2.087890625, -0.08148193359375, -1.1904296875, 0.10443115234375, -0.56201171875, 0.2802734375, 0.409423828125, 0.74365234375, 0.308349609375, -0.86328125, 0.1123046875, -0.82373046875, -0.81298828125, 0.42626953125, -0.44580078125, 0.64599609375, 0.377197265625, 0.396484375, -0.3564453125, 0.5556640625, -0.7265625, -0.5458984375, -1.5712890625, -1.892578125, -0.46484375, 1.013671875, 0.5078125, -0.97119140625, -1.3642578125, -0.1513671875, -0.294189453125, -0.55810546875, 1.0615234375, 1.8154296875, 0.79541015625, 0.0242156982421875, 0.35498046875, 0.6337890625, -1.2373046875, 0.41748046875, -0.921875, 0.70849609375, 0.493896484375, -1.052734375, 1.267578125, 0.544921875, 0.06903076171875, 0.13134765625, -0.630859375, 0.287841796875, 0.6474609375, 0.10418701171875, 1.4140625, -2.525390625, -0.22802734375, 0.48583984375, 2.220703125, 1.6455078125, -0.0548095703125, 1.43359375, 0.5615234375, -1.4990234375, -1.1083984375, -0.41064453125, 1.7138671875, -0.7978515625, -0.939453125, 0.47802734375, 0.6728515625, -0.0177459716796875, 1.0947265625, -0.12042236328125, 1.3046875, -0.276611328125, 0.353271484375, -0.0379638671875, 0.1849365234375, 0.70166015625, -0.409912109375, 1.18359375, 0.29150390625, 0.458984375, 1.482421875, 0.20458984375, 1.27734375, -0.1480712890625, -0.552734375, -0.0169219970703125, 0.2998046875, 1.541015625, -0.7255859375, -0.466552734375, -0.89208984375, 0.66552734375, -0.95703125, 2.44140625, 0.28466796875, 1.39453125, -0.10931396484375, -0.218505859375, -0.46533203125, -2.619140625, -0.343017578125, -0.50048828125, -0.371826171875, -0.88330078125, -1.611328125, 2.306640625, 0.61279296875, 1.318359375, -0.41650390625, -2.6875, 1.1162109375, -0.6865234375, 1.5283203125, 0.2164306640625, -2.00390625, 0.97265625, -1.0380859375, 2.58984375, -1.4228515625, -0.951171875, -1.982421875, 1.1923828125, -0.2125244140625, -1.7197265625, -0.37060546875, 0.2437744140625, 0.412841796875, -0.85205078125, -0.52294921875, 1.837890625, -1.0439453125, 0.366943359375, 1.1767578125, 0.60791015625, 1.07421875, -0.2489013671875, 0.13427734375, -0.4130859375, -0.205078125, -0.88916015625, -0.0762939453125, -1.048828125, -0.5576171875, -1.470703125, 0.2308349609375, 1.966796875, -0.1273193359375, -0.62548828125, 1.7724609375, -2.310546875, -0.497314453125, 2.474609375, 0.142333984375, -0.3955078125, -1.388671875, 0.58203125, 1.18359375, 1.1015625, 1.2275390625, -0.454345703125, -0.6494140625, 1.73046875, 1.3896484375, 0.04833984375, 0.7138671875, -1.3681640625, 1.3544921875, -1.9033203125, 0.130615234375, 1.8935546875, -0.1927490234375, -1.73828125, -0.63720703125, 2.033203125, 0.281005859375, -0.05792236328125, -1.3740234375, 2.14453125, -0.62939453125, 1.6123046875, 1.3115234375, 1.8916015625, 0.4619140625, -0.0151214599609375, 1.02734375, 2.1796875, -0.06939697265625, 0.290771484375, -0.8369140625, 0.72607421875, 1.3212890625, 0.5234375, 1.3759765625, -0.015869140625, -0.3935546875, -0.69873046875, 1.126953125, 0.62890625, -0.12225341796875, -0.1888427734375, 2.083984375, -0.82080078125, 0.72265625, 1.892578125, 0.94189453125, 0.06329345703125, 1.1005859375, -0.08319091796875, 0.8251953125, -0.336669921875, 0.6611328125, 0.7607421875, 0.8544921875, 1.3896484375, -0.64892578125, -0.454833984375, 0.01270294189453125, -0.39208984375, 0.439453125, 0.02471923828125, -0.54248046875, -1.650390625, 0.69091796875, -0.6396484375, 1.1083984375, 0.52197265625, -0.60888671875, -1.8515625, 1.1259765625, -2.05859375, 0.477294921875, -1.7216796875, 1.0302734375, 0.89306640625, -1.4130859375, 1.423828125, 0.298583984375, -1.390625, 0.74609375, 0.841796875, 0.1510009765625, 0.93798828125, -0.91455078125, 0.927734375, 0.316650390625, 1.990234375, -0.1998291015625, -1.44921875, -0.99267578125, -1.0341796875, 1.0556640625, -0.142333984375, -0.0999755859375, -1.208984375, 0.583984375, -0.54150390625, -1.3310546875, 0.31591796875, -0.253173828125, 2.494140625, 1.0791015625, 0.9208984375, 0.409423828125, -0.45947265625, 1.412109375, -1.0517578125, -1.5234375, 0.400634765625, 0.0518798828125, -0.373046875, -2.51171875, 0.7275390625, 0.438232421875, -0.045257568359375, -0.1348876953125, -0.17333984375, -1.4853515625, 0.69384765625, -0.62890625, 1.0322265625, 0.806640625, -0.4091796875, -0.0450439453125, -1.251953125, 0.68212890625, 1.033203125, 0.56298828125, 0.025421142578125, 0.470458984375, 0.425537109375, 0.167724609375, 0.71533203125, 1.6591796875, -0.69580078125, -0.005023956298828125, 0.033660888671875, -0.27001953125, 0.123291015625, 0.86962890625, 0.4658203125, -2.458984375, 2.095703125, 0.6142578125, 0.32421875, 1.7958984375, 2.447265625, -0.56591796875, 0.58056640625, -1.796875, -0.1617431640625, 0.240234375, -0.242919921875, -2.232421875, -0.419921875, -2.708984375, -0.0178070068359375, -0.51416015625, -0.368408203125, 1.6005859375, 0.4619140625, 2.47265625, 1.4892578125, 1.3134765625, 0.849609375, -1.42578125, -1.0439453125, -0.254150390625, 1.298828125, -0.42529296875, 0.7216796875, -0.9404296875, 0.2276611328125, 0.0565185546875, -1.1044921875, -0.3125, 0.86474609375, -0.65185546875, -1.1953125, -2.00390625, -0.619140625, 0.87255859375, -0.415771484375, 1.2216796875, 0.13330078125, 0.9404296875, 0.01702880859375, -0.298583984375, 0.285888671875, -0.42236328125, 1.54296875, -0.494873046875, -1.6611328125, 0.187255859375, -0.61376953125, -0.56494140625, 0.043914794921875, -1.9453125, -0.3017578125, 0.8095703125, 0.72509765625, -1.71875, -0.81298828125, -1.248046875, -0.66162109375, 1.4609375, -0.335693359375, 0.9775390625, 0.27783203125, -0.67138671875, 0.484375, -0.599609375, -0.5556640625, 0.65625, 0.67431640625, -0.64990234375, 1.8349609375, -1.0986328125, -0.1304931640625, -0.431640625, 1.2705078125, -2.505859375, 0.66015625, 0.09576416015625, 0.38037109375, -0.376220703125, -0.53955078125, -0.65625, -0.57470703125, -0.3388671875, 0.52783203125, -0.7021484375, 0.96533203125, -1.41796875, 0.7080078125, -0.37841796875, 1.1923828125, -0.66162109375, -1.70703125, 0.58935546875, 0.443359375, 1.0341796875, 0.2423095703125, -0.168212890625, 0.5869140625, -1.2021484375, 0.9775390625, -0.56201171875, 0.367431640625, 2.880859375, 0.160888671875, -0.1578369140625, 0.2109375, -0.5517578125, -0.158203125, -0.72607421875, -0.476806640625, 1.1015625, 0.58154296875, 0.377685546875, -0.3193359375, -0.376708984375, 0.6240234375, -0.81689453125, -0.55126953125, 0.299560546875, -0.09393310546875, 0.1427001953125, -2.046875, -0.80810546875, 0.58642578125, -0.0789794921875, -0.955078125, -0.51025390625, 0.87158203125, -0.97900390625, -1.6123046875, 0.39306640625, -0.9130859375, 0.693359375, 0.296630859375, -1.259765625, 0.84228515625, 0.87548828125, 1.3720703125, 0.92138671875, 2.341796875, 1.0908203125, -1.921875, 0.08978271484375, -1.306640625, 0.1414794921875, -1.8525390625, -0.439697265625, 0.298095703125, 0.69677734375, 0.92626953125, 0.7998046875, 0.884765625, 0.419677734375, 0.181396484375, 0.708984375, -0.9765625, -0.85400390625, -0.482666015625, -0.5595703125, 0.06390380859375, 0.7421875, -1.033203125, -0.8037109375, 0.6953125, -0.319091796875, -0.43896484375, -0.88037109375, -2.205078125, 0.1336669921875, -0.80517578125, -0.2021484375, 0.5126953125, -0.7333984375, -0.6923828125, -0.1385498046875, 1.25, -1.2001953125, -0.266845703125, -0.223876953125, 0.62841796875, 1.4365234375, 2.17578125, -0.247314453125, 1.6513671875, 0.1619873046875, -0.15869140625, 0.1175537109375, 2.494140625, -0.63232421875, -0.221923828125, -0.90771484375, -0.1966552734375, -1.25390625, 1.458984375, -1.1240234375, 0.84228515625, 0.47802734375, 1.1005859375, -0.296630859375, -0.5576171875, -0.8837890625, -2.244140625, -1.458984375, -1.3076171875, -0.038238525390625, -1.45703125, -0.876953125, 0.7119140625, -1.9248046875, -1.7763671875, -1.796875, -1.9091796875, 0.10455322265625, 2.056640625, -2.708984375, -1.21484375, -1.6845703125, -0.1036376953125, 0.12493896484375, -0.68212890625, -0.1767578125, -0.104736328125, -1.6201171875, 0.9287109375, -1.2314453125, 0.60498046875, -0.132568359375, -0.73583984375, -0.331298828125, 1.408203125, -0.1549072265625, 0.0538330078125, -0.1278076171875, -1.1171875, -1.75, 0.423583984375, 0.51904296875, -1.1474609375, 1.2177734375, -0.2001953125, -0.99267578125, 1.1123046875, -1.001953125, -1.6591796875, -1.0546875, -0.5263671875, 0.442626953125, -0.1658935546875, -0.26025390625, 0.37841796875, -0.97900390625, 0.04779052734375, -1.724609375, -0.255126953125, 1.5830078125, 0.171142578125, 0.3984375, -0.1060791015625, 0.1734619140625, -0.53857421875, -0.44921875, -1.3974609375, -1.029296875, -0.2020263671875, -0.04876708984375, -0.194091796875, 1.1826171875, 1.8173828125, -0.552734375, -0.0054931640625, -0.55908203125, -0.55029296875, -0.0298919677734375, 0.1231689453125, -1.298828125, -0.861328125, 0.3154296875, 0.82080078125, 0.74560546875, -1.0888671875, -0.5009765625, -0.049774169921875, -1.220703125, -0.68505859375, -0.0253753662109375, -0.04547119140625, 0.38818359375, 0.401611328125, -2.462890625, -0.346435546875, 2.1640625, -0.1865234375, 0.67138671875, -0.01113128662109375, -0.34814453125, 1.1806640625, 0.66943359375, -1.06640625, 1.3271484375, 0.98876953125, -0.97607421875, 0.77685546875, 0.2047119140625, 0.3603515625, -0.2041015625, -1.228515625, 0.0506591796875, -0.96923828125, -0.436279296875, -0.890625, 0.047515869140625, 1.2158203125, 0.87255859375, -1.0048828125, 1.4052734375, 0.405517578125, -0.9921875, -0.7275390625, 1.3935546875, -0.0155181884765625, -0.537109375, -0.08294677734375, 1.625, 1.3681640625, 2.38671875, -0.1063232421875, 0.391357421875, 0.7099609375, -0.282470703125, -0.55419921875, -1.11328125, 1.4072265625, 0.453369140625, 1.9091796875, -0.08331298828125, 0.685546875, 0.91259765625, 0.1331787109375, 0.85888671875, 1.81640625, -1.23828125, 0.63671875, -1.5390625, 0.8447265625, 1.3046875, -1.2880859375, 0.61181640625, 0.51611328125, 0.90234375, 0.423828125, -0.1865234375, -1.466796875, 1.1689453125, 1.3232421875, 0.32763671875, 3.134765625, 0.5703125, 1.17578125, 1.5126953125, 1.6728515625, 2.015625, 0.67529296875, 1.4150390625, 0.50537109375, -0.451416015625, 0.17529296875, -0.147705078125, -1.0087890625, 0.7451171875, 0.54736328125, -0.837890625, 0.142333984375, 1.33984375, 1.50390625, 0.587890625, 1.693359375, -0.47021484375, -0.2305908203125, 0.1622314453125, 1.13671875, -0.79296875, -0.053131103515625, -0.62158203125, 0.1876220703125, 0.228271484375, 2.34375, -0.71923828125, -0.247802734375, -1.423828125, -0.82080078125, 0.1558837890625, 0.2066650390625, 1.16015625, 1.390625, 0.84619140625, -1.263671875, 0.49658203125, -1.8759765625, -0.53125, 0.1424560546875, 0.54052734375, -0.2548828125, 0.46142578125, 0.53955078125, -1.359375, -1.02734375, -0.2454833984375, 1.8740234375, 0.13232421875, -1.1728515625, -0.697265625, 0.331787109375, 1.568359375, 1.513671875, 0.50732421875, -0.39794921875, 0.344970703125, -0.69873046875, 1.1357421875, -1.2509765625, -0.07733154296875, 0.289794921875, -1.537109375, 0.5185546875, 1.890625, 0.8955078125, -1.54296875, -0.2286376953125, 0.8818359375, 1.3466796875, 0.5029296875, 1.7958984375, -0.78125, 0.24072265625, 2.15625, -1.39453125, 0.4765625, 0.05841064453125, -0.28955078125, -0.1875, 0.90478515625, 0.748046875, 0.2412109375, -0.45751953125, -1.328125, -0.431396484375, -1.029296875, 1.8134765625, 0.329833984375, -1.3896484375, -0.849609375, -1.2763671875, -1.28515625, -1.263671875, -0.416748046875, -0.306396484375, -1.4951171875, 1.2958984375, 0.83056640625, -1.2841796875, -1.8876953125, 0.73486328125, -1.046875, 1.68359375, 1.5615234375, -1.494140625, 0.66748046875, -0.2021484375, 1.6083984375, -1.640625, 0.861328125, -1.279296875, -0.348388671875, 0.66162109375, 0.302978515625, 0.61962890625, 1.47265625, 0.9296875, -0.6650390625, -0.748046875, -0.037353515625, 1.126953125, 0.06475830078125, 1.490234375, -1.0595703125, -1.2939453125, -0.289306640625, -0.49169921875, -1.634765625, -1.9443359375, -0.451904296875, 0.0623779296875, -0.94970703125, 0.01442718505859375, -2.2734375, -0.35986328125, -0.372802734375, -1.1455078125, -0.95654296875, -0.448486328125, -0.6220703125, -0.466064453125, -0.339111328125, 0.89599609375, -0.88134765625, -0.8857421875, -0.10443115234375, -0.99169921875, 1.5517578125, -1.1064453125, 0.85546875, 1.1201171875, 0.287353515625, -0.27880859375, -0.266357421875, -1.1455078125, -1.0771484375, 0.67333984375, -0.728515625, 0.1627197265625, -0.77783203125, -1.5390625, -0.3212890625, -0.96337890625, -1.0556640625, -1.2412109375, -1.0673828125, -1.03125, 0.061767578125, 0.203857421875, -0.408447265625, -0.525390625, 0.86767578125, 0.57373046875, -1.5234375, 0.83544921875, 0.16796875, -0.59814453125, 0.9853515625, 0.32666015625, 0.72900390625, -1.0234375, 0.284423828125, 0.099853515625, -0.7333984375, 0.495849609375, -0.306884765625, -0.06915283203125, -0.87939453125, -1.1806640625, -0.58154296875, 0.0623779296875, -0.399658203125, -0.677734375, -0.76318359375, 1.46484375, 0.466064453125, -1.138671875, 0.96826171875, -0.2474365234375, 1.720703125, 0.7373046875, 0.495849609375, 0.041412353515625, 0.3740234375, -0.4375, 1.236328125, -0.0182647705078125, -1.0654296875, 0.3974609375, 1.2294921875, -0.397705078125, 0.1927490234375, 0.2069091796875, -0.337890625, 0.284912109375, 0.787109375, 0.5341796875, 1.0283203125, -1.0458984375, -0.007434844970703125, -1.640625, 0.6123046875, 0.38525390625, -0.199951171875, -0.452392578125, -0.300537109375, -0.30419921875, -0.4794921875, -0.60888671875, 0.448486328125, -0.98095703125, -0.96826171875, -0.634765625, 0.7939453125, -0.63232421875, -0.3046875, -0.262939453125, 1.00390625, 3.201171875, 0.161865234375, 0.75341796875, 0.33642578125, 0.08319091796875, 0.030975341796875, 1.0712890625, 1.6396484375, -0.27099609375, -0.669921875, 0.96728515625, -0.08050537109375, 0.5146484375, 0.85107421875, 1.1552734375, 1.5185546875, 2.7890625, 0.165283203125, -0.69677734375, -0.73486328125, 2.4765625, 0.60107421875, -0.303955078125, 0.828125, -0.0828857421875, 0.50732421875, -1.1435546875, 1.064453125, -0.85986328125, 3.05859375, 1.064453125, -0.69189453125, -2.43359375, -1.38671875, 0.57421875, -0.5791015625, 1.748046875, 0.5126953125, 0.6123046875, 0.6357421875, -1.7880859375, 0.939453125, -0.66796875, 0.00699615478515625, 0.2841796875, 0.681640625, 0.008026123046875, 0.4091796875, -0.69580078125, -1.916015625, -1.384765625, -0.4951171875, -1.62890625, -0.87060546875, 0.95556640625, -0.5107421875, -1.9111328125, 0.399658203125, -0.83544921875, 1.4755859375, 1.0361328125, 0.81591796875, -0.056243896484375, -0.814453125, -0.339111328125, 0.1214599609375, 1.19140625, -1.125, 1.7138671875, 0.94580078125, 0.44189453125, 0.1343994140625, 0.73876953125, -1.1748046875, 1.2412109375, 1.580078125, -1.0224609375, -1.5234375, 0.39599609375, 1.451171875, 0.7158203125, 0.378662109375, 1.21484375, -1.8134765625, 0.320556640625, 0.3291015625, -0.71484375, -0.497314453125, -0.1126708984375, -0.5390625, -0.56103515625, 1.287109375, 0.481201171875, -1.64453125, -0.60302734375, -0.25634765625, -0.421630859375, -0.404052734375, -0.99560546875, 0.74755859375, 0.5830078125, -0.004467010498046875, -1.0810546875, -0.458740234375, -0.08135986328125, 0.02618408203125, 1.6123046875, -1.876953125, -0.116455078125, -0.1756591796875, -1.5107421875, 0.256591796875, 1.037109375, 0.022613525390625, -1.62890625, -1.0380859375, 0.08984375, 0.712890625, -0.4736328125, -0.0440673828125, 2.189453125, -0.55908203125, 1.9345703125, 1.37890625, 1.048828125, -0.5498046875, 1.037109375, 2.041015625, -0.57275390625, 0.9072265625, 1.36328125, 0.401123046875, -0.8310546875, -0.2298583984375, 1.0859375, 0.1796875, -0.227294921875, 0.32958984375, -0.442626953125, 1.3408203125, -0.3671875, -1.2705078125, 0.313232421875, -0.496826171875, -0.2198486328125, -0.11865234375, 0.93115234375, 0.1275634765625, 1.466796875, 0.71240234375, 0.1973876953125, 0.77099609375, 0.88623046875, -0.299560546875, -1.3359375, -2.404296875, -0.234375, -0.86474609375, 0.74560546875, 1.353515625, 0.316650390625, -0.62744140625, 0.27197265625, -2.08203125, 1.4462890625, 1.6005859375, -0.04119873046875, -0.56689453125, 1.2529296875, -0.2281494140625, 0.890625, -0.53759765625, 0.533203125, -0.72021484375, -1.8056640625, -0.50341796875, 2.744140625, -0.736328125, 0.4755859375, 1.060546875, 0.329345703125, 0.9189453125, -0.90087890625, -2.392578125, 0.90478515625, 0.1539306640625, -0.018707275390625, -0.151611328125, 1.0087890625, -1.18359375, -0.329345703125, 0.1292724609375, -1.642578125, -1.392578125, 2.17578125, -1.6318359375, 0.415283203125, 0.37109375, -1.2060546875, 1.03515625, -1.6064453125, 0.52392578125, 1.0673828125, -0.74560546875, -0.71435546875, 2.357421875, -0.51513671875, -0.291015625, -0.650390625, -1.2919921875, -0.72216796875, -0.2406005859375, 0.6279296875, 0.05499267578125, 0.68798828125, -1.6318359375, 0.83154296875, -1.453125, 0.35986328125, 0.1805419921875, 0.032440185546875, -0.69677734375, 1.060546875, 1.3828125, 1.1591796875, 0.317626953125, -0.79541015625, 0.2138671875, 0.24853515625, -1.689453125, 1.533203125, -0.2958984375, -1.1796875, -2.6328125, 1.490234375, 0.37890625, 0.319091796875, -0.3193359375, -0.1192626953125, 1.9833984375, 1.0908203125, 0.59521484375, -0.388427734375, 1.33984375, -0.1807861328125, -0.74169921875, -0.1632080078125, 0.99560546875, -1.4755859375, -0.118408203125, -0.147216796875, -0.3369140625, 0.64111328125, 0.97119140625, -0.318359375, -2.54296875, 0.44970703125, 0.70751953125, -0.66259765625, 1.95703125, -0.72900390625, 0.1263427734375, 1.5224609375, 0.134765625, -1.3818359375, 0.54833984375, -0.46875, -0.1097412109375, 0.325439453125, -0.94091796875, -0.39453125, -0.40625, 1.0361328125, -0.01654052734375, -0.26708984375, -0.37744140625, 0.853515625, 0.814453125, 1.142578125, -1.3330078125, 0.783203125, -0.92919921875, -0.98681640625, 1.2236328125, 0.541015625, 1.654296875, -0.67333984375, 0.4814453125, -1.4365234375, 0.017242431640625, 0.466796875, -0.97802734375, 0.1383056640625, -1.068359375, -0.7783203125, -0.1251220703125, 0.77734375, 0.8544921875, -0.61376953125, -0.857421875, -0.090576171875, -1.6962890625, 0.8232421875, -0.435546875, -0.0179290771484375, -0.478271484375, -0.28173828125, 1.4599609375, 1.3408203125, -0.27490234375, 1.24609375, 0.483642578125, 0.0828857421875, -0.43017578125, -2.408203125, -0.9111328125, -1.796875, 0.537109375, 0.345703125, -0.83984375, 1.359375, -0.93115234375, 0.486328125, 0.75830078125, 0.947265625, -0.370849609375, -0.340576171875, 0.161865234375, 0.1351318359375, -1.0859375, 0.8671875, -0.384521484375, 0.2137451171875, -0.2978515625, -1.341796875, -0.59228515625, -1.1826171875, -2.177734375, -0.152587890625, -0.49853515625, 0.83251953125, 0.2093505859375, 0.67578125, -1.8212890625, 1.3388671875, 2.421875, -0.28955078125, 1.142578125, -0.60205078125, -0.794921875, -0.342041015625, -1.3125, 0.95556640625, -0.004669189453125, -1.5634765625, -1.7724609375, -2.40234375, -0.06817626953125, 1.0478515625, -0.75048828125, 0.73388671875, -0.04632568359375, -0.366943359375, -1.40234375, -1.60546875, -1.1005859375, 0.55322265625, 0.087646484375, -0.472900390625, 1.28515625, 0.18408203125, -0.544921875, 2.35546875, -0.70458984375, 0.6806640625, 0.2998046875, -0.10174560546875, 0.38671875, -0.841796875, -1.4443359375, -1.30078125, 0.7099609375, 0.400390625, 0.4501953125, 1.68359375, 0.88427734375, 1.34375, -0.931640625, -0.40673828125, -0.732421875, 0.9287109375, -1.1708984375, 0.78369140625, 0.41748046875, -1.173828125, 0.85107421875, 1.046875, 0.56787109375, -0.260498046875, 0.06671142578125, -0.271484375, 1.6533203125, -1.4833984375, 2.35546875, 0.499755859375, -1.2900390625, -3.353515625, -0.4521484375, 0.23193359375, -0.375244140625, 0.97998046875, 0.4423828125, 0.493896484375, 0.25244140625, 1.8125, 2.703125, 0.0751953125, 0.3935546875, -0.7978515625, -0.73974609375, -1.6640625, -0.05133056640625, -0.0966796875, 0.1812744140625, 1.0986328125, -0.11968994140625, 0.869140625, 0.387939453125, -0.83251953125, 1.0498046875, -0.132568359375, -1.984375, 0.900390625, -0.99853515625, 0.662109375, -1.365234375, 0.235107421875, -0.9111328125, 1.1083984375, -0.609375, -0.442626953125, -2.248046875, 1.056640625, 2.853515625, -1.587890625, 0.60693359375, -2.77734375, 0.2347412109375, -0.484619140625, 0.62353515625, 1.08984375, 0.1240234375, -0.32763671875, -0.185546875, -0.78076171875, 1.3486328125, 0.6767578125, 0.6318359375, 0.76806640625, 0.488525390625, -0.342529296875, 0.100830078125, 0.133056640625, 0.54443359375, -1.654296875, 0.72412109375, 0.459716796875, -2.265625, -0.07598876953125, 0.8134765625, 0.386962890625, 0.5185546875, -1.30078125, -0.2435302734375, 0.685546875, -1.4326171875, -0.76171875, 0.3759765625, 1.3505859375, 0.70263671875, -1.0068359375, -1.0419921875, -0.6357421875, -1.3740234375, 0.40966796875, 0.447509765625, -1.4423828125, 0.330810546875, 0.495361328125, 1.435546875, 0.109130859375, -1.255859375, -0.240478515625, -0.371826171875, -0.67529296875, -0.446533203125, 1.388671875, -0.11376953125, 0.160400390625, -0.158203125, -1.9111328125, 0.806640625, -0.69140625, -0.201171875, 0.99853515625, -1.642578125, 1.109375, 1.1865234375, -0.304931640625, -0.02392578125, -0.67529296875, -1.2685546875, -0.955078125, 1.7001953125, -1.2158203125, 0.259033203125, 1.1787109375, -2.00390625, -0.2166748046875, 0.94140625, 0.484130859375, -0.38525390625, -0.0792236328125, -1.31640625, 2.341796875, -1.744140625, -0.80908203125, 1.7236328125, 2.025390625, 0.34765625, -0.90087890625, -0.411865234375, -0.53662109375, 0.638671875, 1.6728515625, -0.94873046875, 0.6455078125, 1.2021484375, -2.11328125, 0.7861328125, -1.1953125, 0.75048828125, 2.267578125, 0.70458984375, 1.0849609375, -1.2421875, -0.7587890625, -2.0703125, -0.2381591796875, 1.4013671875, -0.58349609375, -2.859375, -0.146484375, 0.350830078125, 1.26953125, -1.427734375, -0.095458984375, 0.7646484375, -0.92919921875, 0.59765625, 0.84619140625, 0.38525390625, 0.043975830078125, -0.51220703125, -0.8544921875, -0.419921875, 1.1962890625, 0.33203125, 2.279296875, 1.615234375, 0.303955078125, 0.0938720703125, -0.62451171875, -1.0869140625, 0.08807373046875, -1.16796875, 0.8759765625, 0.2529296875, -0.270751953125, -2.1796875, 0.1461181640625, -0.60205078125, 0.330078125, 0.1072998046875, -0.27001953125, 0.81982421875, 0.44921875, -0.6416015625, 0.29150390625, -1.109375, -0.7021484375, -1.5205078125, 2.060546875, 0.693359375, -1.5205078125, 0.9482421875, 0.496826171875, 0.501953125, -0.7255859375, -1.177734375, 0.75634765625, 1.3251953125, 1.1220703125, 1.751953125, -0.595703125, -0.18994140625, 0.326904296875, 0.5458984375, 0.9482421875, 0.505859375, -0.472900390625, -1.8056640625, -0.509765625, -0.2021484375, 0.6044921875, -1.0166015625, -1.267578125, 1.796875, -2.08984375, -0.72509765625, -0.457275390625, 0.499267578125, -1.5078125, 1.0498046875, -1.6455078125, 0.9111328125, -0.474365234375, 1.888671875, 0.3671875, -0.0027790069580078125, -1.091796875, -1.2099609375, 0.229736328125, 0.1441650390625, 0.865234375, 0.487060546875, -0.9228515625, 1.0458984375, -0.11761474609375, -1.052734375, -0.37646484375, 1.1181640625, 2.9296875, 0.1270751953125, 0.400146484375, 1.6044921875, 0.462158203125, 0.3583984375, -1.32421875, 0.4404296875, 0.736328125, 0.95654296875, -0.56689453125, 0.61279296875, -0.35498046875, -0.8798828125, -0.34912109375, 0.896484375, -1.3505859375, 1.169921875, -0.232421875, 2.029296875, 1.396484375, -1.7333984375, -0.93408203125, -0.8828125, -0.80615234375, -1.1943359375, -0.3125, -0.81005859375, -0.2025146484375, 0.90673828125, -1.556640625, -0.460693359375, 0.420654296875, -0.59765625, 0.50390625, -0.10772705078125, 0.07073974609375, -0.82763671875, 2.59765625, -0.51025390625, 0.430908203125, 0.415771484375, -0.5927734375, -0.65478515625, 0.56103515625, 0.4072265625, 1.3408203125, 1.091796875, -0.4931640625, -1.1513671875, 0.1656494140625, -2.9453125, 0.358154296875, 1.841796875, 1.1083984375, 0.39892578125, -0.02117919921875, -2.29296875, 0.63427734375, -0.97021484375, 0.1141357421875, -0.0679931640625, 0.703125, 0.379638671875, -0.31494140625, 0.48779296875, -0.1846923828125, 0.5390625, 0.56982421875, -0.2431640625, -0.0908203125, -1.8828125, -2.078125, -2.19140625, -1.681640625, 1.86328125, -1.115234375, -0.45458984375, 0.265625, -1.4423828125, -1.109375, 1.818359375, -0.1964111328125, -0.26318359375, -1.0498046875, -1.263671875, -1.556640625, -0.59814453125, 0.06951904296875, -0.1622314453125, 0.69482421875, -0.400390625, -0.09954833984375, 0.29638671875, -0.476318359375, 0.74755859375, -0.6923828125, -1.01171875, 1.0361328125, 0.30322265625, 1.9208984375, 0.75634765625, 1.1865234375, -0.057342529296875, -0.58447265625, 0.73486328125, 0.267578125, 2.162109375, 0.6435546875, -0.442626953125, 0.55615234375, -0.1251220703125, 0.8232421875, 0.09954833984375, 0.6572265625, 2.298828125, -0.16943359375, 0.28271484375, -0.296875, -0.043487548828125, 1.1201171875, -1.396484375, 0.235595703125, -1.6201171875, -1.0673828125, 1.3935546875, 1.3046875, -1.763671875, -0.61328125, -1.150390625, -0.84765625, 0.385009765625, -0.09246826171875, -0.68310546875, 0.288818359375, -1.298828125, 0.76171875, -0.5166015625, 0.320068359375, -0.3388671875, -1.564453125, -0.487548828125, 0.107666015625, 0.94091796875, 0.18115234375, 0.935546875, -1.681640625, -0.423583984375, -0.408203125, 1.4111328125, 1.38671875, -0.548828125, -0.95703125, -0.7060546875, -0.40966796875, 0.42333984375, 0.59130859375, -0.233642578125, 1.4765625, 0.626953125, 0.5146484375, 1.8486328125, -1.8603515625, -1.89453125, -0.748046875, -0.300537109375, 0.62548828125, 0.1597900390625, 2.515625, 1.1435546875, 0.90869140625, 0.8271484375, -1.9736328125, 0.140869140625, -0.1151123046875, 0.82470703125, 0.99658203125, -1.109375, 0.32373046875, 0.94970703125, -1.2265625, 0.68408203125, -1.296875, -2.134765625, -0.15869140625, 0.66015625, 0.131103515625, 1.6904296875, 1.130859375, -0.91796875, 0.23974609375, 0.10284423828125, -0.84375, 0.1781005859375, 0.54150390625, 0.1878662109375, 0.388427734375, 1.013671875, 1.24609375, 1.392578125, 1.876953125, 1.1435546875, -2.220703125, -0.198974609375, -0.51416015625, 0.52880859375, 1.341796875, -0.52734375, -1.798828125, -0.6728515625, 0.1658935546875, 0.62255859375, -0.92724609375, -3.06640625, -0.34423828125, 1.412109375, -1.314453125, -1.6279296875, 0.1868896484375, -2.68359375, 1.0029296875, -0.58544921875, 0.18310546875, -0.00893402099609375, 1.1171875, 1.9501953125, 1.1103515625, 0.312255859375, -1.892578125, -0.2744140625, 2.267578125, -0.9873046875, 0.70947265625, -0.564453125, 0.252197265625, 1.1494140625, -0.66162109375, -1.1923828125, 0.60009765625, -0.7265625, 1.724609375, 2.822265625, 0.4658203125, -1.95703125, -1.6240234375, 0.435546875, -0.180908203125, 0.736328125, 0.287353515625, -0.171142578125, -0.21240234375, 0.01043701171875, -0.314208984375, -0.4931640625, 0.1629638671875, -0.447265625, -0.193603515625, 1.43359375, 2.013671875, 1.666015625, -1.7783203125, -0.7333984375, 0.83544921875, -0.10418701171875, 0.8232421875, 0.054931640625, -1.146484375, -1.341796875, 1.525390625, -1.134765625, 1.2802734375, 0.67626953125, 1.1611328125, 0.927734375, -0.12335205078125, -0.302978515625, -0.17236328125, 0.61279296875, -0.18359375, 0.48046875, 0.306396484375, 0.92919921875, 0.1591796875, 0.5830078125, -1.9775390625, 0.022613525390625, 1.078125, -1.2021484375, 1.7646484375, -0.1876220703125, -2.125, 0.404541015625, -0.76513671875, -1.296875, -1.205078125, 0.94873046875, -0.74853515625, -0.9853515625, 0.9912109375, 1.0517578125, 0.415771484375, 0.92529296875, -0.22314453125, -0.273193359375, -0.343017578125, -1.3203125, -1.0224609375, 0.421142578125, -1.1181640625, 0.671875, -0.67626953125, 2.3984375, -0.436279296875, 0.0689697265625, -1.44140625, 0.6796875, -1.181640625, 0.090087890625, 0.83642578125, -0.467041015625, -0.236083984375, 1.6796875, 0.1827392578125, 0.2391357421875, 0.1832275390625, -0.75537109375, 1.90234375, -0.734375, 2.09375, 0.11480712890625, -0.05078125, 1.59375, -2.294921875, 0.0799560546875, 0.0257720947265625, 2.2578125, 0.556640625, -0.200927734375, 1.8701171875, 1.044921875, 1.0537109375, -0.326904296875, -1.595703125, 0.0279541015625, -1.3369140625, -0.67333984375, 0.2186279296875, -0.1578369140625, -0.97314453125, 1.0859375, -2.580078125, 0.250732421875, 0.93017578125, 1.533203125, 1.2158203125, -0.400146484375, 0.7763671875, -1.6904296875, 0.0772705078125, 0.966796875, 0.900390625, -1.6357421875, 0.368896484375, 0.0750732421875, 0.230224609375, -0.82958984375, -1.837890625, -2.052734375, -0.1756591796875, -0.59814453125, -2.828125, -2.013671875, 0.84033203125, 1.759765625, -0.78271484375, -0.392822265625, 0.4619140625, -0.438232421875, -0.69287109375, -0.77490234375, -1.1201171875, 2.279296875, -1.30859375, -1.3154296875, 0.2362060546875, -0.67626953125, 1.923828125, 1.0869140625, -0.52734375, -0.5693359375, 0.2890625, 0.07452392578125, -0.97998046875, 0.1641845703125, -0.314208984375, -0.465576171875, -0.324951171875, 0.049468994140625, 0.1680908203125, 0.80908203125, -1.3984375, 0.078125, -0.73779296875, -2.173828125, -0.268310546875, 1.98046875, -0.06982421875, -2.31640625, 0.93505859375, -0.315673828125, -1.146484375, -0.5205078125, 1.0380859375, -0.52587890625, -0.244873046875, -0.6015625, -1.2333984375, -0.10968017578125, -1.4580078125, -0.91259765625, -1.408203125, -2.021484375, -1.6748046875, -0.6923828125, 1.3330078125, 1.1025390625, -0.57568359375, 0.015472412109375, -0.419189453125, 1.169921875, 0.395263671875, -1.6455078125, -0.01505279541015625, -0.60400390625, -0.52587890625, 0.41943359375, 1.0908203125, 0.165771484375, 0.1517333984375, -0.583984375, 1.58203125, 1.08203125, -0.5791015625, 1.1904296875, -0.8623046875, -0.60546875, -0.650390625, -1.3759765625, 0.779296875, 0.73779296875, 0.298828125, -0.364013671875, -1.05078125, -0.343505859375, -1.712890625, -0.467529296875, 0.391845703125, 0.1552734375, 0.87548828125, 1.3369140625, 0.99072265625, 0.463623046875, -1.0498046875, -0.505859375, 0.99755859375, 0.75927734375, 0.42919921875, 1.1826171875, -0.603515625, 0.86865234375, -0.607421875, -1.443359375, -2.083984375, -1.669921875, -0.332275390625, -0.30859375, -0.4296875, -0.5888671875, 1.3642578125, 1.869140625, 2.609375, 1.171875, -1.4111328125, -1.416015625, -1.4873046875, 0.70263671875, -0.57470703125, 0.88525390625, 0.6494140625, -2.134765625, 0.2310791015625, 1.4189453125, -1.4365234375, -0.11968994140625, 1.1044921875, -0.46923828125, 1.1025390625, 0.08966064453125, -1.0068359375, 0.779296875, 0.0201568603515625, -0.5986328125, -1.43359375, 1.13671875, -1.275390625, -0.841796875, -0.333251953125, -0.60986328125, 0.9453125, 1.490234375, 0.83203125, -1.296875, -0.049072265625, -1.064453125, -0.308837890625, 0.5654296875, 1.7998046875, 1.103515625, -0.67919921875, 0.08624267578125, 1.111328125, -0.163330078125, -0.1451416015625, 0.7177734375, -0.59033203125, -0.61474609375, -1.2119140625, 0.0694580078125, 0.1888427734375, -0.353271484375, -1.0908203125, -1.4619140625, -0.159912109375, -0.2303466796875, 0.08062744140625, -1.228515625, 0.97509765625, -0.61279296875, 0.55908203125, 0.146240234375, -0.9697265625, -1.220703125, 0.3671875, 0.15380859375, -0.285888671875, -0.783203125, 0.59521484375, 0.787109375, -1.591796875, -1.19140625, 0.64306640625, 0.04296875, 0.1146240234375, -0.216552734375, 1.0751953125, -0.6669921875, -1.3203125, -1.1279296875, 1.162109375, 1.4189453125, -0.9580078125, -0.62841796875, 0.51318359375, -0.066650390625, -1.0439453125, -0.323486328125, 1.2021484375, -1.935546875, -1.119140625, -1.7294921875, 0.1988525390625, -1.17578125, -0.76025390625, 1.9716796875, -0.48486328125, -3.421875, 1.228515625, -1.4326171875, 0.63427734375, 0.53466796875, -0.61962890625, -1.7431640625, 0.076904296875, 0.1807861328125, -1.068359375, 0.386474609375, -0.2646484375, 2.07421875, 1.310546875, 1.837890625, 0.2203369140625, 0.33251953125, -0.31005859375, -0.71044921875, -0.49267578125, -0.379150390625, -1.4404296875, -0.35498046875, -0.35986328125, 1.189453125, 0.10382080078125, 0.3681640625, -1.3408203125, 0.280517578125, -0.376220703125, -0.9873046875, 0.6865234375, 0.810546875, 1.15625, -0.88671875, 0.5361328125, 0.08575439453125, 1.166015625, 0.39501953125, 0.03704833984375, 0.2005615234375, -2.333984375, -0.443359375, 2.51171875, 1.12890625, -0.60888671875, 0.471923828125, -0.85205078125, 1.1630859375, 1.25390625, -1.8544921875, -0.275634765625, -1.1416015625, -0.82568359375, 1.787109375, -1.0576171875, 1.09765625, 0.8330078125, 0.36865234375, -2.068359375, -1.103515625, -1.5, -0.04302978515625, -0.74072265625, 1.0849609375, -0.1884765625, -0.64306640625, -0.55810546875, -0.76513671875, 1.5546875, 0.896484375, -0.705078125, -0.314453125, 0.2156982421875, -0.2237548828125, 0.89306640625, -0.76904296875, 1.2646484375, -1.5556640625, 1.1845703125, -0.66357421875, -0.113525390625, 0.607421875, -0.533203125, 0.31103515625, 0.91455078125, -0.666015625, 1.359375, -0.0836181640625, -0.457763671875, -0.95849609375, 0.8466796875, -0.48681640625, 1.0390625, -2.140625, 0.05352783203125, 0.484375, -1.1328125, -0.06329345703125, -1.7451171875, -0.90673828125, 1.302734375, -0.010528564453125, -0.60009765625, 0.3046875, 0.3193359375, -0.2197265625, 1.513671875, 1.025390625, -1.033203125, 0.85205078125, 0.09979248046875, -0.115478515625, -2.3046875, 0.0513916015625, 0.44873046875, -0.61181640625, 0.6416015625, -0.60107421875, -0.9697265625, -0.59228515625, 0.2841796875, -0.34326171875, -2.091796875, 1.4990234375, 1.1279296875, -1.1826171875, 0.85791015625, 1.4716796875, -0.7685546875, -1.326171875, -0.1295166015625, -0.70361328125, -0.474609375, 0.8642578125, -0.0955810546875, 1.34765625, 1.2080078125, 1.2744140625, 0.03759765625, -0.093017578125, 0.224853515625, -1.048828125, -0.67529296875, 2.44921875, -0.728515625, -0.587890625, 1.470703125, 0.70751953125, -1.2607421875, -0.21484375, 0.05218505859375, 0.91845703125, -0.89306640625, 0.6748046875, -1.32421875, 0.01105499267578125, -0.93310546875, 0.28955078125, 0.90771484375, 1.5439453125, 1.27734375, 0.310546875, 0.759765625, 1.0703125, 0.150390625, 0.128662109375, 0.335205078125, -1.9365234375, -0.73876953125, -0.145263671875, -0.681640625, -0.346435546875, -1.26953125, -0.8876953125, -0.7734375, -1.55859375, -0.144287109375, 0.1416015625, 1.384765625, -1.3837890625, -0.369873046875, -0.58544921875, -0.47314453125, -0.96728515625, 0.470947265625, -0.7998046875, 0.413818359375, 0.024139404296875, 0.91357421875, -0.57568359375, -0.046630859375, -0.225830078125, -0.92333984375, 0.3056640625, 0.134765625, -0.458984375, -2.134765625, 0.9482421875, -0.66796875, 1.4072265625, 0.050689697265625, 2.494140625, -1.099609375, 0.099365234375, 0.0166778564453125, 0.447998046875, -0.5361328125, -0.43408203125, -0.1552734375, 0.57568359375, 0.3701171875, -0.445556640625, 0.1453857421875, -0.1878662109375, -2.1796875, -0.325927734375, -1.9677734375, -0.057037353515625, -0.2119140625, 0.49072265625, -0.106689453125, 0.1436767578125, -0.51220703125, -1.0859375, -0.7470703125, -0.716796875, -0.1539306640625, -0.053070068359375, -0.88818359375, 0.169189453125, 1.0068359375, 0.91259765625, -0.146728515625, 0.85498046875, 1.501953125, 0.4580078125, -0.1746826171875, -0.708984375, -1.732421875, -0.74560546875, -1.171875, 0.310302734375, 0.9580078125, 0.389892578125, 0.2364501953125, 0.2354736328125, -0.1385498046875, -0.7197265625, 0.37841796875, -0.318603515625, 0.8564453125, -1.212890625, -2.1171875, 0.403564453125, 0.060272216796875, 0.791015625, -1.0078125, 1.4111328125, -0.51025390625, -1.4482421875, 0.86376953125, -1.2900390625, 0.84033203125, -0.9404296875, 1.7099609375, -0.97998046875, -0.61767578125, 0.88330078125, -0.71142578125, 0.2274169921875, 0.447998046875, 1.244140625, 0.58984375, -0.196044921875, -1.7626953125, 1.091796875, -0.80078125, -0.32080078125, -1.4921875, -0.1148681640625, -1.8115234375, 0.8408203125, -0.75048828125, 0.345703125, -1.5625, -0.2393798828125, -0.6904296875, 0.7666015625, 0.8037109375, -1.6015625, 0.086669921875, 0.6240234375, -0.59765625, -0.005207061767578125, -1.2548828125, -2.474609375, 1.35546875, 0.49462890625, -0.85693359375, -0.2091064453125, 0.477783203125, 0.0181884765625, 1.4287109375, -0.362548828125, -0.265625, -0.8896484375, 0.13916015625, 0.27783203125, -0.7080078125, 0.478515625, 1.5390625, -0.96044921875, 1.83984375, 0.40283203125, -0.1455078125, 0.253173828125, -0.1534423828125, 0.98779296875, -1.79296875, -1.5107421875, 0.7919921875, 0.328857421875, 1.6611328125, -0.9892578125, 0.37890625, 1.47265625, -0.8017578125, -0.48828125, 0.650390625, -0.25, -1.271484375, 0.35595703125, 0.57470703125, -0.0022792816162109375, 0.2196044921875, -0.9775390625, 0.151123046875, 1.6259765625, -0.032928466796875, 0.31640625, -0.466064453125, -0.8623046875, -0.216552734375, 1.8271484375, -0.91650390625, -0.55517578125, 0.91943359375, -0.2093505859375, 0.67919921875, 2.734375, 1.6201171875, -0.238037109375, 0.411865234375, 0.2144775390625, 0.4267578125, 0.1904296875, -0.281982421875, -0.33251953125, -0.013763427734375, 0.59130859375, 0.20361328125, 0.1529541015625, 0.1864013671875, 0.8193359375, -0.282470703125, -0.374267578125, 1.580078125, 1.056640625, 1.126953125, -1.626953125, 1.6748046875, -1.248046875, 0.243896484375, 0.25341796875, -1.138671875, -0.428466796875, -0.333984375, -0.166015625, -0.1864013671875, -0.72900390625, 1.3837890625, -0.97607421875, 0.1304931640625, 0.89697265625, -0.1207275390625, -0.301025390625, 0.73486328125, -1.4501953125, 0.83935546875, 0.8857421875, -0.109375, 0.055267333984375, -0.287353515625, -0.97509765625, -0.78076171875, 0.6865234375, 0.56298828125, 2.05078125, -0.515625, 0.9326171875, 1.3603515625, 1.630859375, 0.90234375, -1.5361328125, 1.07421875, 0.40771484375, 0.73486328125, -1.2333984375, -1.3837890625, -1.0791015625, 0.14404296875, 0.398193359375, 0.6669921875, 0.44091796875, 1.3369140625, -1.8310546875, 0.81298828125, -2.544921875, -1.7119140625, -0.1856689453125, -0.047821044921875, 0.4091796875, 1.4853515625, 0.66162109375, -0.6455078125, -0.734375, 1.71875, 0.33251953125, 0.603515625, -0.94921875, 0.6044921875, 0.6201171875, -0.052886962890625, 0.446044921875, 0.445556640625, -0.11651611328125, 1.7431640625, 2.47265625, 0.95361328125, 0.38330078125, 0.85205078125, 0.58203125, -1.0126953125, 0.1373291015625, -0.456298828125, -0.476806640625, 0.98876953125, -0.43359375, -0.841796875, -0.7880859375, -0.59765625, -2.05078125, -0.38134765625, -0.228515625, -0.286865234375, -0.1873779296875, -2.146484375, 0.09991455078125, 0.79736328125, -0.27880859375, 0.489013671875, -0.9052734375, -0.58203125, -0.69140625, 2.060546875, 0.24755859375, 0.35498046875, -0.62255859375, 0.0875244140625, 0.474609375, -0.058013916015625, -1.8173828125, -0.7509765625, -1.04296875, -0.93603515625, 1.0947265625, 2.083984375, -0.529296875, 0.203125, 0.275146484375, 0.0282135009765625, 0.40869140625, 0.280029296875, -0.38037109375, -0.1373291015625, 0.039764404296875, 0.92626953125, -0.317138671875, -2.80859375, 1.013671875, 0.7138671875, -2.708984375, 0.168701171875, 0.80078125, -0.09906005859375, -0.2100830078125, 0.7744140625, 0.32861328125, 1.126953125, 0.9013671875, 0.48974609375, 1.2001953125, 2.01953125, -0.1085205078125, -0.5087890625, 1.4228515625, -1.0009765625, -0.7919921875, 0.6015625, -2.177734375, -0.7978515625, 0.04278564453125, 0.806640625, -0.9658203125, -0.018890380859375, 1.392578125, -0.1861572265625, -1.2314453125, 1.2158203125, -1.5107421875, 0.7861328125, -0.90478515625, 1.7626953125, -0.413330078125, 0.313720703125, -1.8798828125, -0.294677734375, 0.5361328125, -0.7216796875, 0.95556640625, -0.0130157470703125, 0.74609375, 0.384521484375, 1.2783203125, -0.390869140625, 1.3984375, -0.43017578125, 0.372802734375, -1.6259765625, -0.071044921875, 0.279296875, 0.64599609375, 0.149169921875, 0.80810546875, 0.2188720703125, -0.26953125, 0.42724609375, -0.591796875, -0.156005859375, -0.91015625, 0.5341796875, 0.99462890625, -2.36328125, -0.7880859375, 0.7607421875, 0.8125, 0.96435546875, -0.168701171875, -0.446533203125, 1.2275390625, -0.56298828125, -1.57421875, -0.07806396484375, -0.1793212890625, 1.25390625, 0.705078125, 0.476806640625, -0.9296875, 0.9921875, -1.3818359375, -0.2410888671875, 0.36474609375, 0.249267578125, 0.33837890625, 1.544921875, -0.061614990234375, 0.724609375, 2.212890625, -0.654296875, 0.3447265625, -0.339111328125, -0.0341796875, -0.21142578125, 0.09771728515625, 1.724609375, 0.5400390625, 1.328125, 0.53955078125, 0.48828125, 1.4814453125, -0.158935546875, 1.1494140625, 0.1759033203125, 1.54296875, -1.1279296875, -1.927734375, 0.380126953125, 0.185302734375, 0.1068115234375, -0.59716796875, -0.044189453125, 0.67578125, 0.5654296875, 1.005859375, 1.65625, 0.20947265625, -1.2578125, -0.433349609375, 0.380615234375, 0.473876953125, -0.498291015625, 0.09417724609375, -1.072265625, 1.1005859375, -1.0458984375, 0.358154296875, 0.1624755859375, -0.7001953125, -1.2626953125, -0.456298828125, -0.875, -0.338623046875, -0.1968994140625, 1.978515625, 1.669921875, -0.279541015625, 1.236328125, 0.08447265625, 1.3701171875, -0.603515625, -0.480712890625, 0.348388671875, 0.6650390625, 0.8955078125, -0.344970703125, 0.78125, 0.35693359375, 1.2294921875, -0.71923828125, -1.1767578125, -2.0078125, -0.693359375, 1.8447265625, 0.8916015625, 1.4033203125, 1.5302734375, 1.6328125, -1.3603515625, 1.1806640625, -0.375, 0.218017578125, -0.0069732666015625, 0.1607666015625, 0.966796875, -0.295166015625, 0.85400390625, -0.30908203125, 0.9521484375, -0.98828125, 1.1318359375, 0.274658203125, -0.1448974609375, 0.666015625, 0.49755859375, 1.2607421875, 0.81982421875, 1.0302734375, 1.181640625, 1.1298828125, 0.049652099609375, -1.0537109375, 0.2412109375, -0.135009765625, -0.9189453125, 0.41943359375, -0.04931640625, 0.01528167724609375, -0.53857421875, -0.491455078125, 0.320556640625, -0.263916015625, 1.244140625, -0.338623046875, 0.82958984375, -0.52294921875, 0.49462890625, -0.8369140625, 0.71826171875, 1.7548828125, -1.404296875, -1.787109375, 1.0, -0.1109619140625, 1.2109375, -1.5654296875, 0.7783203125, 0.70263671875, -0.40283203125, -0.09271240234375, 0.387939453125, -0.70361328125, -0.52001953125, 1.056640625, -1.927734375, 0.12158203125, 0.032196044921875, -0.365478515625, -0.71044921875, -1.9892578125, -0.1827392578125, 0.275146484375, -0.0200958251953125, 0.712890625, -0.415283203125, 0.82861328125, 1.169921875, -2.365234375, -0.830078125, 0.49267578125, -0.1817626953125, 0.06976318359375, -1.6728515625, -1.2158203125, -0.443359375, 0.73486328125, -0.8662109375, 0.67431640625, 0.2103271484375, 1.701171875, 0.72802734375, 2.037109375, -1.3330078125, 0.41015625, -0.607421875, 1.1845703125, 0.89794921875, -0.433349609375, 1.2109375, -1.296875, -0.81201171875, 1.017578125, 0.81201171875, 1.8623046875, 1.837890625, -1.03125, 0.73681640625, -0.126220703125, 0.191650390625, -1.396484375, 1.966796875, 1.3173828125, -0.63818359375, -0.748046875, 0.62451171875, -1.591796875, 0.310546875, 1.2607421875, -0.09326171875, 1.2783203125, -0.040985107421875, -0.2008056640625, -1.0234375, -0.281494140625, 1.0, 2.787109375, -0.104736328125, -1.4912109375, 0.3310546875, -0.1790771484375, -0.30810546875, 1.15234375, 0.50439453125, 1.912109375, 0.43017578125, 0.84326171875, -0.2454833984375, -0.4345703125, -0.431396484375, 0.97265625, -0.30126953125, 0.98291015625, -0.49658203125, 1.30078125, 1.474609375, 1.4599609375, -1.1640625, -0.1651611328125, 2.109375, -0.494384765625, -1.220703125, 0.439453125, 2.025390625, -1.9384765625, -1.150390625, 0.0870361328125, -1.8642578125, -0.468994140625, 0.89892578125, 0.411376953125, 2.458984375, -0.53662109375, -0.7412109375, 1.1806640625, 0.33251953125, 0.6728515625, -0.1507568359375, 0.25390625, -0.7626953125, 0.208740234375, -0.42236328125, -0.347900390625, 0.79736328125, 0.6826171875, -0.1546630859375, -2.126953125, -1.2431640625, -0.10186767578125, -0.91650390625, 1.5322265625, -2.412109375, -0.57861328125, 0.2529296875, 0.041046142578125, -0.1500244140625, -1.236328125, 0.4716796875, 0.6279296875, 0.42529296875, 1.2021484375, 0.64404296875, 0.7841796875, -0.2484130859375, -1.3134765625, 0.9560546875, 0.459228515625, 0.779296875, -0.8330078125, -0.360595703125, -0.71533203125, -0.95263671875, 0.83544921875, -1.11328125, 0.2376708984375, -0.34375, 0.826171875, 0.8935546875, -0.11138916015625, 0.7509765625, -0.326416015625, 0.201171875, -1.2705078125, 0.9609375, -0.369384765625, -0.397705078125, 0.5546875, -0.192138671875, -0.52587890625, -0.52734375, -0.0080413818359375, -0.72265625, -0.87255859375, -0.798828125, -0.10968017578125, -0.00670623779296875, 1.478515625, -0.62646484375, 1.7919921875, -0.1689453125, -0.5068359375, -0.96435546875, 0.178955078125, 1.2138671875, 0.34716796875, -0.1837158203125, -0.0181427001953125, 0.71728515625, 1.560546875, -1.55859375, -0.5771484375, -1.330078125, -0.71337890625, -0.403564453125, 0.6171875, 0.144287109375, -1.87890625, 1.2880859375, -0.1474609375, 1.5361328125, -0.26708984375, 1.5673828125, 0.070068359375, -0.78271484375, 0.3935546875, -0.673828125, -0.2406005859375, -1.1044921875, 1.3427734375, -0.19921875, -0.484375, 1.1767578125, 0.59814453125, 0.0457763671875, -1.0087890625, -1.4658203125, 1.671875, -1.12109375, -0.6318359375, -0.4052734375, 0.86865234375, 0.24072265625, 1.439453125, 0.005157470703125, -1.609375, 0.41796875, 2.4609375, 2.03125, -1.2646484375, 0.75244140625, -0.03607177734375, -0.732421875, -0.6708984375, -1.388671875, -0.98828125, -0.6025390625, -0.0787353515625, -0.3427734375, -0.10650634765625, -1.7490234375, -0.74267578125, 0.84814453125, 0.3857421875, 0.92919921875, -0.336181640625, -0.1966552734375, -0.2197265625, -0.0239105224609375, 0.77294921875, 0.048370361328125, 0.11474609375, -1.625, 1.123046875, 2.21875, -0.05279541015625, -1.2265625, -0.188720703125, -0.1983642578125, 0.6376953125, -0.328857421875, -1.91015625, 0.304443359375, -1.275390625, 0.81787109375, -0.377685546875, -0.41015625, 0.87744140625, 0.109130859375, -0.642578125, -0.35888671875, -1.08984375, 1.3212890625, 0.56494140625, -0.086181640625, 0.44873046875, -0.7978515625, 1.6650390625, 0.029144287109375, 0.5869140625, -1.1318359375, 0.469482421875, 0.70703125, 0.1304931640625, 0.1168212890625, -0.283935546875, -0.468505859375, -1.33203125, 1.1884765625, -0.10931396484375, 0.2164306640625, 1.20703125, -0.6220703125, 0.68310546875, 0.311279296875, -1.30859375, 0.09869384765625, -0.10833740234375, -1.259765625, -0.64208984375, -1.1201171875, 0.9052734375, -0.61279296875, -1.63671875, 0.447021484375, -0.73388671875, -1.9580078125, -0.038421630859375, 0.498291015625, 0.08819580078125, 0.64208984375, -0.002490997314453125, -0.9716796875, -0.97265625, 0.541015625, 0.3251953125, 1.1513671875, 0.431396484375, -1.2919921875, 0.40283203125, 0.54638671875, -2.25390625, -0.1756591796875, -0.04962158203125, -1.779296875, 1.0478515625, 0.342529296875, -0.724609375, -0.488525390625, 1.1376953125, -0.63134765625, -2.859375, 0.634765625, -0.7265625, -0.257568359375, -0.43115234375, 0.56298828125, -0.38623046875, -0.385498046875, -1.91796875, -0.90869140625, -1.8486328125, -1.892578125, -0.8447265625, -1.001953125, -0.288330078125, -0.71142578125, -0.35302734375, -1.2880859375, -0.31884765625, -0.91357421875, -0.279296875, -1.849609375, -1.505859375, -1.2294921875, 0.81640625, -0.57177734375, -0.376708984375, 0.417236328125, -1.5869140625, -0.365234375, 0.7626953125, -1.357421875, 0.1544189453125, -0.0936279296875, -0.051666259765625, -2.208984375, 0.91650390625, -0.27099609375, 1.33984375, -0.1265869140625, -0.7841796875, -0.058013916015625, -0.595703125, 0.1029052734375, -1.08203125, 0.10809326171875, 0.1363525390625, -1.095703125, -1.5673828125, 1.00390625, -0.599609375, 0.5166015625, 0.4052734375, 0.296142578125, 0.45703125, -1.21484375, -0.1962890625, 0.3037109375, 0.6962890625, -0.771484375, -0.1014404296875, -2.294921875, 1.818359375, 0.6162109375, 0.99267578125, 2.189453125, 1.1103515625, 0.1458740234375, -1.140625, 0.0904541015625, 0.264404296875, 1.5, 0.763671875, -0.32373046875, 0.302734375, 1.228515625, -0.1658935546875, 0.5068359375, -1.5791015625, -0.0163116455078125, -2.15234375, 0.5419921875, 0.77001953125, -0.2191162109375, -0.95166015625, -0.5478515625, 0.2034912109375, 0.0280303955078125, 1.40234375, 0.62548828125, 0.144287109375, 1.6513671875, 0.515625, -0.3955078125, -0.76025390625, 1.546875, -0.416015625, -0.10821533203125, -0.21142578125, 0.17724609375, -1.5322265625, 0.51025390625, 0.5283203125, -0.342041015625, -1.2412109375, -1.41796875, 1.009765625, -1.1962890625, 1.0185546875, 0.058807373046875, -0.055084228515625, 0.81982421875, 0.3349609375, -2.11328125, -1.396484375, 1.7900390625, 0.92138671875, 0.326171875, -0.1490478515625, -0.78125, -1.376953125, -1.44140625, -0.61572265625, 0.0164031982421875, -1.365234375, -0.204345703125, -1.0478515625, -2.080078125, -0.459228515625, 1.197265625, 1.318359375, -0.55615234375, 1.0908203125, -0.6708984375, 1.509765625, -1.0830078125, 0.81005859375, -1.3134765625, 0.97900390625, -0.7255859375, -0.486083984375, 0.78564453125, -0.552734375, -0.70361328125, -0.91015625, -0.6650390625, 1.1025390625, -1.4228515625, 0.262451171875, -0.36962890625, -0.041168212890625, -0.74658203125, 0.859375, 1.1865234375, -0.9140625, -0.69287109375, -2.607421875, -2.046875, -1.1376953125, -0.53564453125, -0.66162109375, -0.21044921875, 0.129638671875, 1.1337890625, -0.136474609375, -0.78466796875, 1.3095703125, 0.3037109375, 0.045501708984375, -1.3388671875, -0.042694091796875, 0.59033203125, 0.5986328125, -0.483154296875, -1.8349609375, -0.10009765625, -1.0732421875, -1.80859375, 0.405029296875, -1.2421875, 0.04071044921875, -2.173828125, 0.54296875, -0.1978759765625, -0.0625, 1.4765625, 0.23828125, 0.0015172958374023438, 0.68212890625, -0.2093505859375, -0.014556884765625, -1.1279296875, 0.69921875, 1.5498046875, -1.419921875, 0.189697265625, -0.6259765625, 0.76025390625, -0.241943359375, -0.60009765625, 0.0987548828125, -0.75537109375, -1.1171875, 0.3671875, -0.2978515625, 0.73486328125, -0.44580078125, 1.0791015625, -0.328369140625, -0.16943359375, -0.73388671875, -1.2041015625, 0.44580078125, -0.798828125, -0.276123046875, 0.7607421875, -0.452392578125, 0.0670166015625, -1.1572265625, -0.537109375, -0.275146484375, 0.63427734375, 0.343505859375, 0.51904296875, 0.98291015625, 0.0791015625, -0.65771484375, 1.0576171875, -0.42236328125, -0.404296875, -1.6591796875, -0.06390380859375, 0.6376953125, -1.74609375, 0.716796875, -0.9423828125, 0.3193359375, -0.5966796875, 0.142578125, -1.962890625, -1.4892578125, -2.67578125, -0.3994140625, -0.316162109375, 1.220703125, 1.6337890625, 0.37841796875, -1.56640625, -1.2607421875, 1.6962890625, -0.269287109375, -0.80029296875, 0.90625, -0.29638671875, -0.47900390625, -0.494140625, 0.08697509765625, 2.595703125, -0.372314453125, -2.37109375, 0.34375, -0.31787109375, -0.51025390625, -0.68701171875, -0.27685546875, 0.31005859375, 0.215576171875, 0.6337890625, 0.393310546875, 0.638671875, 0.5859375, -0.7060546875, 0.343017578125, -0.1470947265625, 0.326904296875, 1.9892578125, 1.1591796875, 1.9130859375, -1.181640625, -0.291748046875, 0.8046875, -1.1806640625, -1.166015625, -2.052734375, 1.3916015625, -1.8125, 0.70947265625, 0.73388671875, -0.529296875, -0.6220703125, 0.8154296875, 0.8701171875, -1.0791015625, 0.3544921875, 0.72412109375, -0.8515625, -0.494873046875, 0.4775390625, -0.0026912689208984375, 0.00832366943359375, -1.73828125, 0.435546875, 0.619140625, 1.5, 0.1944580078125, -1.30078125, 1.5537109375, 0.98486328125, -0.1368408203125, 0.6748046875, -0.80810546875, -1.533203125, -0.8056640625, -1.83984375, 0.31298828125, -0.07550048828125, -0.5263671875, -0.48095703125, 0.499755859375, -0.5107421875, -0.158447265625, 1.3369140625, -0.88818359375, -0.1239013671875, 0.87060546875, -1.2880859375, -0.9189453125, -0.357666015625, 1.84765625, -0.0208892822265625, 0.74365234375, 0.904296875, -0.97998046875, -1.9814453125, -0.030242919921875, 0.57421875, -0.90625, 0.47802734375, -0.51123046875, 0.1337890625, 0.5654296875, -0.486572265625, 1.1982421875, -1.2412109375, -1.849609375, -0.29296875, -1.8818359375, 1.9169921875, -0.8671875, -1.001953125, -2.138671875, 2.126953125, 0.69287109375, 0.5244140625, -0.76416015625, -1.482421875, 0.7568359375, 0.82177734375, -0.20263671875, -0.55712890625, 1.0078125, -0.67578125, -1.3369140625, 0.98681640625, 0.361083984375, -0.9365234375, -0.4287109375, 1.234375, -0.036285400390625, -1.462890625, 0.4794921875, -0.43603515625, -1.9111328125, 0.60205078125, -0.50732421875, -0.10797119140625, 0.0169830322265625, 1.244140625, 1.1044921875, 0.33251953125, -0.488525390625, -0.5615234375, 0.02703857421875, 0.4091796875, -0.49267578125, -0.448486328125, -0.70458984375, 1.908203125, -0.73876953125, -0.0390625, 0.53857421875, -0.304931640625, 0.049285888671875, -0.89990234375, 0.47021484375, -2.283203125, 0.11260986328125, -0.412109375, -1.11328125, 2.2265625, 0.80859375, 1.0791015625, 0.40625, 0.163818359375, 0.032379150390625, -0.05328369140625, 0.3154296875, -0.26513671875, 0.92578125, -1.001953125, -0.7607421875, -0.10601806640625, 0.39794921875, -1.775390625, 1.2333984375, 0.01605224609375, 1.70703125, 0.078369140625, -1.8388671875, -2.10546875, -1.193359375, 0.134033203125, -0.46533203125, -1.03125, -0.931640625, 0.3388671875, -1.150390625, -1.373046875, 0.268798828125, 0.7666015625, -1.56640625, 0.0212249755859375, 0.1751708984375, 0.0909423828125, 0.9833984375, 0.274658203125, -0.02252197265625, 0.50244140625, 0.266357421875, 0.5830078125, 1.3115234375, -0.07061767578125, -0.005756378173828125, 0.2493896484375, 1.3681640625, 0.390625, -0.052764892578125, -0.2607421875, 1.4296875, 0.53173828125, 0.5712890625, -0.0240936279296875, -0.5947265625, -2.28515625, -0.96875, -0.8271484375, 1.8798828125, -0.76220703125, -0.435546875, 0.53857421875, -1.369140625, -1.728515625, -1.8291015625, 1.197265625, -0.354248046875, 1.1181640625, -1.58203125, 2.09375, 1.5859375, 0.1815185546875, -0.58984375, -1.0673828125, 0.48388671875, -1.4755859375, 0.63720703125, 0.32177734375, -0.57080078125, -1.0859375, -0.1544189453125, 0.2197265625, -1.37890625, 0.55078125, -1.8173828125, 2.466796875, 0.8955078125, 0.3564453125, 0.414306640625, -0.2452392578125, -0.2230224609375, -0.09173583984375, -0.70166015625, -0.17236328125, 0.6953125, 1.306640625, 0.5810546875, -1.8583984375, 0.236083984375, -0.91845703125, 1.54296875, 0.20458984375, -1.650390625, -0.8701171875, 0.272216796875, -1.0322265625, -0.42919921875, -0.85498046875, -1.41796875, -0.96044921875, 0.54248046875, -2.634765625, 0.314208984375, -0.293701171875, -1.388671875, 0.364501953125, -1.25, 0.373046875, -1.1533203125, -1.033203125, 0.1614990234375, -0.7236328125, 0.192626953125, 0.78173828125, 0.02655029296875, 0.12255859375, 1.025390625, 0.5224609375, -1.7421875, 0.06536865234375, -0.640625, -0.828125, -1.68359375, 1.6083984375, 0.457763671875, 0.62158203125, -0.66650390625, 0.54345703125, -0.8935546875, -1.62890625, -1.4453125, -0.07830810546875, 0.1573486328125, 0.6962890625, -0.1307373046875, 0.39208984375, 0.178466796875, 0.9228515625, 3.158203125, 1.3955078125, -0.9609375, 0.68212890625, 1.115234375, -0.56005859375, -0.06500244140625, -0.231201171875, 0.1578369140625, 0.5234375, 0.85888671875, -0.294189453125, -0.5400390625, -0.4423828125, 1.84375, 1.15625, 0.0684814453125, 0.54052734375, 0.1212158203125, -0.59423828125, 0.6025390625, -1.302734375, -0.55224609375, -1.0703125, -0.262451171875, 0.26220703125, -2.388671875, -0.3974609375, 0.293701171875, -0.402587890625, -0.5322265625, -0.213134765625, 1.4453125, -1.4072265625, -0.442138671875, -0.5068359375, -1.9609375, 0.73876953125, 0.224365234375, 0.71484375, -0.3564453125, 0.51123046875, 0.80517578125, -0.62109375, 0.1982421875, -1.0205078125, -0.3359375, -0.78173828125, 1.3505859375, 0.260986328125, -0.35009765625, -1.2314453125, 0.73583984375, -0.33984375, -0.73291015625, 1.3076171875, -0.1412353515625, -1.5869140625, 1.6552734375, -0.400634765625, 0.3466796875, -0.376953125, 0.33935546875, -0.923828125, -1.818359375, -1.0146484375, 0.36279296875, 0.114013671875, -0.62841796875, -1.2568359375, -0.43896484375, 3.0517578125e-05, -0.80126953125, 0.452880859375, -0.291748046875, -0.9091796875, 0.98583984375, -0.3779296875, -0.60595703125, -0.76904296875, -0.73486328125, -0.8740234375, 1.009765625, 1.517578125, -0.07049560546875, 0.556640625, 0.2454833984375, 1.3564453125, 0.6630859375, -0.64404296875, -0.55517578125, -0.2213134765625, 0.301025390625, -0.5771484375, -1.1787109375, 0.73681640625, -2.857421875, -0.7353515625, -1.0517578125, -0.0582275390625, 1.958984375, -0.382568359375, 1.23046875, 0.232421875, 0.87939453125, -1.34765625, 0.58056640625, -2.31640625, -1.7265625, -0.0017843246459960938, -0.1644287109375, -1.9306640625, 1.609375, 0.08673095703125, -0.6162109375, 0.2119140625, -1.419921875, -0.52587890625, -0.266357421875, 1.2724609375, -0.164794921875, -0.55810546875, 0.263427734375, 1.546875, 0.330322265625, -0.11810302734375, 0.7216796875, 0.2056884765625, 0.2198486328125, -0.88427734375, 0.31298828125, -1.482421875, -1.5498046875, -0.85595703125, -0.1744384765625, -0.9375, -0.51025390625, 0.49267578125, 1.6943359375, -0.0165557861328125, 0.34423828125, 0.56884765625, 0.70751953125, -0.1907958984375, 0.6318359375, 0.448974609375, -0.87939453125, -0.68896484375, -0.55322265625, 2.3125, 0.38525390625, 0.71044921875, -0.958984375, -0.42626953125, -0.275634765625, -0.85595703125, -0.53369140625, 0.31787109375, -0.265869140625, 0.97412109375, 0.734375, 0.40478515625, 0.759765625, 0.3642578125, -0.13134765625, 0.658203125, -0.331787109375, -0.04132080078125, 0.9697265625, 0.3271484375, -0.225830078125, -0.666015625, -1.5927734375, 1.0380859375, 0.404052734375, -0.08416748046875, -1.7255859375, -2.220703125, 0.85400390625, 2.205078125, 0.6044921875, 0.82275390625, -1.3369140625, 0.0810546875, 0.377197265625, -1.123046875, -0.1707763671875, 4.01171875, -1.9765625, -0.07421875, 0.11614990234375, 0.10467529296875, 0.07208251953125, 0.283447265625, -0.307373046875, -1.009765625, -0.86962890625, -1.041015625, -0.869140625, 0.52490234375, 1.5400390625, -0.5087890625, -0.1455078125, 0.246337890625, -0.81982421875, -0.5986328125, 0.218994140625, 1.111328125, -0.7568359375, 0.78271484375, 2.21875, 0.0626220703125, 0.2177734375, -0.55810546875, -1.048828125, -0.4443359375, 0.064453125, -0.6015625, -0.94580078125, -0.94482421875, -0.53857421875, 0.95263671875, 1.060546875, -0.417236328125, -0.045501708984375, -0.736328125, -0.69873046875, 0.55810546875, -0.98291015625, -0.58056640625, 0.72607421875, -0.17578125, -0.334716796875, -0.697265625, -0.4462890625, 0.273193359375, 0.0728759765625, 0.48779296875, 0.0009713172912597656, -0.340087890625, -0.27490234375, -0.1478271484375, 0.454833984375, -0.197021484375, -1.8017578125, 0.51220703125, 1.7470703125, -1.27734375, 0.3359375, -0.1907958984375, -0.2086181640625, 0.67919921875, -0.03533935546875, 0.24755859375, -0.11138916015625, 0.76318359375, -0.5927734375, 1.5009765625, -1.1357421875, 2.759765625, -1.2265625, -0.398193359375, 0.0283966064453125, 1.4423828125, 0.888671875, 1.3701171875, -0.3701171875, 1.396484375, 0.75830078125, -0.6865234375, -0.6064453125, -0.537109375, -1.5634765625, -0.90625, -1.1611328125, -1.58203125, 1.013671875, 0.68359375, -0.385498046875, -0.6845703125, -0.6611328125, -1.6962890625, -2.56640625, 2.33984375, 0.0889892578125, 0.75634765625, -0.8271484375, -0.289794921875, 0.418701171875, -0.069580078125, 0.09564208984375, 0.97705078125, -0.291015625, -0.58154296875, 1.9775390625, 0.10028076171875, -0.042022705078125, -1.9697265625, -0.59765625, -0.92578125, 0.1136474609375, 0.468505859375, -0.046875, 1.4111328125, 2.03125, -3.111328125, 0.06402587890625, 0.607421875, -0.263427734375, 0.69091796875, 2.572265625, 1.0283203125, -1.5224609375, 0.064208984375, -1.25390625, 0.9931640625, -0.0220184326171875, -1.6259765625, -0.0567626953125, -1.4423828125, -2.677734375, 1.146484375, 0.54345703125, 1.365234375, -1.515625, -0.708984375, 0.1549072265625, 0.326171875, 0.0166778564453125, 0.385986328125, 0.478271484375, 0.865234375, 0.97705078125, -1.828125, -1.71875, -0.19384765625, -0.3505859375, -1.2373046875, -0.498779296875, 2.78515625, -0.6708984375, -0.79248046875, 0.14990234375, 0.767578125, -0.82177734375, 0.405517578125, 0.8916015625, -2.00390625, -0.9052734375, -1.3037109375, 0.38623046875, -0.1673583984375, -0.316162109375, 0.20751953125, -0.251953125, 0.8271484375, -0.802734375, -0.484619140625, -0.4462890625, -3.15625, -0.414306640625, 0.6748046875, -1.5283203125, 1.1962890625, 0.1898193359375, -1.24609375, 1.12890625, 0.277099609375, -0.54296875, -1.638671875, 0.65380859375, 0.338623046875, -0.50634765625, 0.984375, 0.47607421875, -0.417236328125, -0.64697265625, 0.76708984375, 1.8701171875, 0.14794921875, 1.48046875, -0.99365234375, -0.56103515625, -0.0810546875, -0.1968994140625, 0.28857421875, 0.65087890625, -0.354736328125, 0.105712890625, -0.01250457763671875, 0.9462890625, 0.04248046875, 0.1551513671875, 0.38525390625, 0.09344482421875, -0.95849609375, 0.73974609375, -0.65673828125, 0.58837890625, -0.0826416015625, 0.305908203125, 0.87353515625, 0.000690460205078125, -1.1962890625, 0.837890625, 0.85986328125, 2.126953125, 0.75634765625, 1.318359375, -1.00390625, 0.1583251953125, -1.064453125, -1.837890625, -0.80029296875, -0.34716796875, -1.6630859375, -0.59521484375, -0.67333984375, 0.12005615234375, 0.07635498046875, -0.1607666015625, 0.40869140625, 0.2403564453125, -0.67529296875, -0.787109375, 1.16015625, 0.75341796875, -0.416015625, -2.1875, -0.34033203125, -0.5634765625, 0.55859375, 1.02734375, -1.2529296875, 1.21484375, -0.77978515625, -0.06463623046875, 1.1767578125, -0.84765625, 0.411376953125, -0.59228515625, -0.0889892578125, -1.328125, 0.9794921875, 0.177001953125, 1.1796875, -0.58447265625, 0.253662109375, -0.15087890625, 0.1671142578125, 0.70947265625, -0.437255859375, -1.107421875, -0.92626953125, -0.58984375, 0.7529296875, -0.962890625, -1.3056640625, -0.018798828125, 0.35107421875, -2.705078125, 0.2467041015625, 0.11322021484375, -2.21484375, -2.208984375, 0.822265625, -1.0087890625, -1.021484375, 0.178955078125, 0.71826171875, -0.2998046875, -1.41796875, 0.465576171875, -0.68896484375, 0.419921875, -0.779296875, 0.146728515625, -0.970703125, -0.447998046875, -1.283203125, 0.52880859375, -0.53076171875, 0.3984375, 0.5068359375, 0.421142578125, -0.462646484375, -1.078125, -0.30908203125, -0.310302734375, 0.298095703125, -0.42919921875, -0.1197509765625, 1.3447265625, 0.25390625, -1.7333984375, -0.62060546875, -0.619140625, -0.358154296875, -1.4169921875, -1.2841796875, 0.07098388671875, -0.20458984375, 2.6015625, -0.3134765625, 1.705078125, -1.2275390625, 1.1923828125, 0.09344482421875, 0.5458984375, 0.2080078125, -0.85302734375, 0.3125, 0.380859375, -0.96142578125, 0.67919921875, -1.3984375, -1.7041015625, 0.060455322265625, -0.71923828125, 0.14208984375, -0.8515625, -0.36669921875, -0.33349609375, -0.408447265625, -0.357421875, 0.497802734375, 1.6572265625, -0.919921875, 0.85546875, -1.0927734375, 0.20703125, 0.583984375, 0.9951171875, -0.66162109375, -0.67431640625, 0.137939453125, -1.4462890625, 1.7841796875, 0.10174560546875, 0.347412109375, -1.123046875, -0.734375, -1.3583984375, -0.381103515625, -2.068359375, 0.363525390625, -0.537109375, -0.2098388671875, 1.046875, -0.05596923828125, 1.1923828125, -1.423828125, -0.220458984375, 1.220703125, -2.375, -0.091064453125, 0.04541015625, 1.1796875, -0.270751953125, -0.08984375, 0.30810546875, 0.53125, -0.96337890625, -2.326171875, -0.2342529296875, 0.81494140625, 1.3662109375, 0.4111328125, 0.587890625, -1.46875, 1.130859375, -0.0013256072998046875, -0.463134765625, 0.42578125, -2.92578125, -0.662109375, -0.424072265625, -2.1171875, 0.9873046875, 1.8349609375, -0.33935546875, -0.9443359375, 0.421142578125, -0.000823974609375, 1.15234375, -0.294189453125, -0.2900390625, 0.50146484375, 0.474365234375, -0.28662109375, -0.343994140625, 0.72265625, 0.4375, 0.81689453125, -0.401611328125, -0.3232421875, 0.4677734375, -0.1251220703125, -1.93359375, -1.2939453125, -0.0179595947265625, -3.140625, 0.337646484375, -1.0283203125, -0.286865234375, -1.1044921875, 2.76171875, 0.76318359375, -2.193359375, -0.29248046875, 0.228759765625, 0.0112152099609375, 1.009765625, 1.8466796875, -0.28369140625, -1.45703125, 0.62744140625, -0.0758056640625, -1.2265625, -0.6181640625, 0.49267578125, -0.57080078125, -0.705078125, 1.5703125, 0.352783203125, -0.9619140625, -1.091796875, -1.4111328125, 0.07342529296875, -0.35595703125, 1.466796875, -1.6337890625, -0.2144775390625, 1.3046875, -0.83935546875, 0.646484375, 1.4677734375, -1.1318359375, -0.48193359375, 1.08984375, 0.1181640625, -0.7568359375, -0.00909423828125, -1.6337890625, -1.3115234375, 2.396484375, -0.05389404296875, -0.94677734375, 0.9619140625, -0.68798828125, 0.18310546875, 0.0552978515625, -0.37060546875, -1.08203125, -0.6103515625, -0.36572265625, 0.39013671875, 1.3955078125, -0.44482421875, -0.415771484375, -0.61669921875, 0.92333984375, -0.0430908203125, 1.1494140625, -0.658203125, 0.278564453125, -2.53515625, -0.66650390625, 0.4794921875, -0.06536865234375, 1.142578125, -0.97216796875, -0.84375, -0.039581298828125, 1.1279296875, 1.6123046875, 1.4091796875, -0.141845703125, -1.6337890625, -0.5634765625, 1.3173828125, -1.0458984375, 0.66796875, 0.496826171875, 0.2000732421875, -1.1201171875, 1.322265625, -1.6181640625, -1.6533203125, 0.5986328125, 0.544921875, -0.82666015625, 1.111328125, -0.0166778564453125, -0.8857421875, 0.2685546875, -0.8955078125, 1.6298828125, 0.271728515625, -1.3115234375, -1.525390625, 0.3896484375, -1.4443359375, 0.00992584228515625, -0.0233306884765625, 0.2286376953125, 0.476806640625, 0.56689453125, -0.480224609375, 0.362548828125, 1.3720703125, -0.3349609375, -0.71240234375, 1.2099609375, -1.1435546875, -1.6650390625, 0.0005097389221191406, 0.459716796875, 0.116943359375, 0.77978515625, 0.350341796875, 0.343994140625, -1.0966796875, 3.439453125, 0.73876953125, -1.068359375, -0.76708984375, -0.033294677734375, -0.2054443359375, -0.4990234375, 0.751953125, 1.142578125, 1.8681640625, -0.9384765625, -0.07159423828125, 0.1064453125, -1.5107421875, 1.0205078125, 0.306884765625, 1.2666015625, 0.239013671875, 0.74365234375, -0.02685546875, 0.775390625, -0.51708984375, -0.54736328125, -1.00390625, 0.1268310546875, -0.294189453125, 0.378662109375, -1.171875, -0.0239410400390625, -2.09765625, -0.2261962890625, -0.80078125, 0.0692138671875, -0.44287109375, 0.89111328125, 1.1884765625, -0.11065673828125, 0.65478515625, 1.0205078125, -1.0390625, -0.396728515625, -1.19140625, -2.0078125, -0.81298828125, -0.8212890625, -0.66259765625, -0.54248046875, -2.396484375, 0.8544921875, 0.332275390625, 0.685546875, -1.2890625, -0.2783203125, -1.2822265625, 0.73486328125, 0.892578125, 1.578125, -0.57080078125, 1.4287109375, 0.342041015625, -0.326904296875, 0.54296875, 3.06640625, -0.027557373046875, -1.2060546875, 0.798828125, -0.66015625, -0.515625, 0.4453125, 0.51806640625, -0.79638671875, 1.4970703125, 0.99169921875, -0.136474609375, 0.63232421875, 1.7177734375, 0.14892578125, -0.67041015625, 0.95263671875, -0.48876953125, -0.2171630859375, 0.12939453125, -1.142578125, -1.2421875, -0.033905029296875, 1.2490234375, 0.260986328125, -1.4921875, 0.36083984375, -0.1712646484375, 1.337890625, 1.2138671875, -1.0185546875, -1.40234375, 0.83740234375, -0.23046875, -0.71142578125, -1.943359375, -0.2890625, -2.166015625, 0.638671875, 0.05853271484375, 0.1494140625, 0.703125, 0.381591796875, -0.65576171875, -1.359375, 0.5283203125, -0.345703125, -0.78759765625, 1.0830078125, -2.5703125, 0.62255859375, -0.78515625, -1.466796875, 0.396484375, 0.53759765625, -0.360595703125, -0.192626953125, 0.08001708984375, -0.409912109375, -0.0765380859375, -0.935546875, 0.90673828125, 0.50390625, -1.306640625, 0.9814453125, -2.109375, -1.2197265625, 0.8955078125, -0.163330078125, 1.4296875, 0.06982421875, 1.7900390625, -2.189453125, -0.9375, -1.056640625, 0.5048828125, 2.001953125, -1.2197265625, 2.71484375, -0.004627227783203125, -0.456787109375, -1.4404296875, 1.201171875, -0.2822265625, -0.2113037109375, -0.65966796875, 1.5087890625, 0.2978515625, -1.8359375, -1.2333984375, -0.468994140625, -0.44970703125, -1.06640625, -2.05078125, -0.408447265625, -0.46923828125, 1.5791015625, 0.48583984375, 1.41796875, 0.98046875, -0.188720703125, 0.72021484375, -0.888671875, 0.6318359375, 0.170166015625, -0.21728515625, 0.646484375, 0.87158203125, -1.05078125, 0.052520751953125, 0.07904052734375, -1.552734375, 0.1484375, -0.312744140625, -0.9765625, -0.630859375, -0.2010498046875, -0.38134765625, -1.3759765625, -0.5986328125, -1.3310546875, 1.26953125, -1.716796875, -0.28369140625, -0.09869384765625, 0.58349609375, -1.640625, -0.25830078125, 0.80126953125, -0.330810546875, -1.0703125, 0.9345703125, -1.3046875, 1.1787109375, -0.157470703125, 0.389404296875, 0.197021484375, 0.720703125, -0.1312255859375, 1.1005859375, -1.12109375, 0.42333984375, 0.56884765625, -0.262939453125, -0.383056640625, 0.595703125, 0.71337890625, 1.888671875, 0.5498046875, 0.2498779296875, -1.2802734375, -0.5888671875, -0.115478515625, -0.7861328125, -1.3369140625, 0.23291015625, -0.548828125, 0.01187896728515625, 0.72705078125, -1.0859375, 0.91455078125, 0.6533203125, 1.2646484375, -0.6650390625, -1.115234375, 0.08319091796875, 1.6826171875, 0.83447265625, 1.0087890625, 0.95654296875, -0.84521484375, 1.1083984375, -0.249755859375, -0.27392578125, -1.052734375, 0.6162109375, -1.5888671875, -0.1806640625, -0.8955078125, -0.3525390625, -2.0546875, 0.7373046875, 0.57470703125, -0.13916015625, -0.394775390625, -0.221923828125, -0.93310546875, 0.14501953125, 0.328125, 0.34814453125, 0.97998046875, -0.48095703125, 1.3017578125, -1.1416015625, 1.1435546875, 0.264404296875, 0.56591796875, 0.7421875, 0.623046875, -1.3623046875, -0.79345703125, 1.59765625, -0.82763671875, 0.83056640625, -0.263671875, 0.07373046875, -0.8125, 0.238037109375, 0.051483154296875, -0.66845703125, -0.80029296875, -0.880859375, 0.56640625, -0.7880859375, 1.3251953125, -0.7060546875, -0.485595703125, -0.331298828125, 1.4482421875, 0.55078125, -0.146728515625, 0.96875, 1.71875, -1.134765625, 1.05859375, -2.5859375, 0.59228515625, -0.36962890625, -0.76171875, -0.1922607421875, 0.6708984375, 0.15673828125, 0.97509765625, 0.39794921875, -0.43701171875, 0.38818359375, -0.2861328125, -0.91015625, 0.0926513671875, -0.82763671875, -1.037109375, 0.1627197265625, 0.51171875, -0.7216796875, -1.1591796875, -0.3095703125, 0.291748046875, -0.482421875, 0.9375, 2.24609375, 0.279052734375, -0.36474609375, -0.02227783203125, -0.74853515625, -1.1767578125, 0.08917236328125, 1.6796875, 1.0185546875, 0.349853515625, 0.433837890625, 0.3837890625, 0.93603515625, 1.2216796875, -1.3798828125, -0.29931640625, 0.7548828125, -0.1065673828125, -0.433837890625, -0.1328125, -0.564453125, 1.4892578125, -1.24609375, -0.99755859375, 0.88037109375, 0.2822265625, 0.06988525390625, -0.9169921875, -0.59228515625, 2.2578125, 1.033203125, -0.826171875, 0.3359375, -0.92333984375, -0.2069091796875, 1.37890625, -0.80322265625, 0.56640625, 1.0771484375, -1.12890625, -0.4169921875, -1.56640625, 0.857421875, 0.3271484375, -1.0615234375, -1.763671875, -0.8271484375, 1.3935546875, 0.740234375, 0.07049560546875, 1.0224609375, -0.54833984375, 0.2115478515625, -0.81787109375, -1.6474609375, -0.6474609375, 1.0302734375, 0.1505126953125, -0.98486328125, -0.6357421875, 0.325439453125, -1.1513671875, 1.1787109375, 0.53857421875, 0.1678466796875, -0.69873046875, -0.1182861328125, -0.2274169921875, -1.099609375, 0.07061767578125, -0.59521484375, 0.1390380859375, 1.1083984375, 0.662109375, -0.6923828125, 0.357666015625, -2.400390625, -0.67529296875, 0.364990234375, -0.1544189453125, -1.7451171875, -0.041351318359375, 0.5673828125, 0.057525634765625, -0.3857421875, 1.8408203125, 0.1927490234375, 1.390625, -0.708984375, -0.82080078125, -0.1671142578125, -0.268798828125, 0.296142578125, 0.5986328125, 1.224609375, 1.1455078125, -0.3828125, -0.022674560546875, 2.17578125, -1.69140625, -0.759765625, -0.021575927734375, -0.5009765625, 0.89697265625, -0.943359375, 1.0322265625, -0.25830078125, 1.0283203125, 0.0259857177734375, -0.56396484375, 0.005023956298828125, -0.047760009765625, -1.2529296875, -1.5126953125, -1.537109375, 0.300048828125, -1.58203125, 0.86376953125, -1.658203125, -0.82177734375, 0.70068359375, -0.53173828125, -1.4619140625, -0.34814453125, 0.91015625, 1.0908203125, 1.263671875, -0.4765625, 0.489013671875, -0.68310546875, -0.337646484375, -0.83935546875, 1.0126953125, -0.9921875, 0.457763671875, -0.08721923828125, -1.587890625, -0.321533203125, -0.767578125, 0.1968994140625, -0.552734375, 0.37353515625, -0.982421875, 0.578125, 2.201171875, 0.541015625, 0.4755859375, 1.3974609375, 1.505859375, 1.3134765625, 0.46337890625, 0.93212890625, 1.7802734375, 0.8984375, 0.14697265625, 2.443359375, 0.2247314453125, -0.043548583984375, 0.483154296875, -0.7841796875, 0.9873046875, -0.779296875, -0.65283203125, 0.29638671875, -1.3017578125, 0.68896484375, -0.310546875, 0.50048828125, 1.2783203125, -0.376220703125, 1.193359375, -0.19775390625, 1.2314453125, 0.89208984375, -0.83251953125, -1.0947265625, 0.46630859375, 0.63330078125, 0.437744140625, -0.07574462890625, -1.3212890625, -2.42578125, -1.6025390625, -1.1064453125, 1.6689453125, 1.0673828125, -0.49951171875, 0.4833984375, 0.48095703125, -1.083984375, -0.06158447265625, 1.24609375, 1.232421875, -1.404296875, 0.16357421875, 0.25830078125, -1.302734375, -0.66015625, 0.57373046875, 0.0026531219482421875, -0.52490234375, 0.58544921875, 0.31689453125, 0.1759033203125, -0.6650390625, 1.0849609375, 0.69775390625, -0.384521484375, -1.4453125, 1.3115234375, -0.2130126953125, 1.228515625, 0.755859375, 0.439208984375, -0.9130859375, 1.060546875, 0.6640625, -0.07525634765625, -2.080078125, 0.251708984375, -1.1689453125, 0.1605224609375, 1.8623046875, 0.08428955078125, 0.421142578125, -0.056427001953125, 1.2900390625, -1.0107421875, 1.0537109375, -1.6904296875, 1.15234375, 0.261474609375, -1.3564453125, -1.3681640625, 0.89111328125, 1.818359375, 0.71875, 1.5263671875, 0.156494140625, 0.86376953125, -0.134521484375, 0.68359375, 0.406982421875, -1.2646484375, 0.5546875, 0.89404296875, -0.2393798828125, -0.8701171875, -0.067138671875, -0.0191650390625, -0.91943359375, 1.1669921875, -0.336669921875, 0.321533203125, -1.5126953125, -0.11151123046875, -0.5322265625, -0.33056640625, 0.8828125, -0.84423828125, -1.5322265625, -0.89501953125, -1.1533203125, -0.291259765625, -0.394775390625, 0.2646484375, 1.4306640625, -2.134765625, 1.736328125, -0.09991455078125, 1.0263671875, 0.031402587890625, -2.84765625, 0.300048828125, 0.27099609375, 0.265625, -0.8212890625, -0.73388671875, 0.30615234375, 2.134765625, -0.68017578125, -0.052703857421875, -1.037109375, 0.433837890625, -0.1978759765625, 0.64453125, -1.5009765625, -1.5244140625, 0.186767578125, -1.37109375, 0.9345703125, 0.42333984375, 0.51806640625, 1.00390625, 2.025390625, -0.251708984375, 0.8095703125, -1.4404296875, -0.1309814453125, 1.7548828125, -0.1192626953125, 0.380615234375, -0.355224609375, -1.998046875, 0.18408203125, -1.1142578125, -1.3076171875, 0.5927734375, 0.88525390625, 0.1900634765625, 2.193359375, -0.310302734375, -1.818359375, 0.283203125, 0.845703125, -1.3125, 0.9501953125, -1.7900390625, 0.63623046875, -1.2412109375, -1.0947265625, 0.732421875, 1.5947265625, -0.71728515625, -0.250732421875, -0.95166015625, 0.031219482421875, 0.304443359375, 0.2032470703125, 0.269775390625, -1.064453125, -1.3896484375, -0.03759765625, -0.44189453125, -1.2529296875, 0.654296875, -0.112548828125, 1.162109375, -0.2171630859375, 0.73974609375, 0.313232421875, -0.1322021484375, 0.77685546875, 1.029296875, -1.103515625, -1.552734375, -0.452392578125, 1.2275390625, -0.421630859375, -0.2100830078125, 0.10980224609375, 0.5693359375, -0.234619140625, -0.482666015625, -0.267333984375, 0.279052734375, -1.4521484375, -0.250732421875, -0.4921875, -1.146484375, 0.0150146484375, 0.8896484375, 0.69140625, 2.0, -0.95263671875, -0.52490234375, -0.013427734375, 0.289306640625, 1.2255859375, -1.556640625, -0.93408203125, -0.2255859375, 1.056640625, -0.0213165283203125, 0.1256103515625, 0.004596710205078125, -1.2373046875, -0.50830078125, -0.236572265625, -0.12548828125, 0.93408203125, 0.0235595703125, -1.033203125, 0.315185546875, 1.05078125, -0.89111328125, 0.475341796875, -0.055419921875, 0.0180816650390625, 0.34765625, 0.413330078125, 1.568359375, 1.5068359375, -0.70947265625, 0.73486328125, -0.69873046875, -0.82861328125, 0.333251953125, 0.43994140625, 1.1171875, 0.93505859375, -0.436279296875, -1.31640625, -0.61279296875, -0.96923828125, -0.055419921875, -1.291015625, 0.37109375, -1.4267578125, 0.52197265625, 0.759765625, 0.60302734375, -1.208984375, -0.293212890625, 1.033203125, -0.77099609375, -1.0341796875, -0.4248046875, 1.294921875, 0.52392578125, -0.03106689453125, 1.4248046875, -1.3662109375, -1.25390625, -0.40625, -0.155517578125, 0.494384765625, -0.51806640625, -0.432861328125, -0.3515625, -0.5283203125, 0.19921875, 0.09686279296875, -0.83544921875, 0.83447265625, 1.6162109375, 0.29443359375, 0.80859375, 0.045623779296875, -0.3828125, 0.482666015625, -0.1829833984375, 0.401123046875, -1.1669921875, 0.734375, -1.5029296875, 0.493896484375, -0.4208984375, -0.90771484375, -1.7138671875, -0.3017578125, -1.2109375, -2.248046875, 0.0755615234375, -0.47802734375, 0.4951171875, -0.37451171875, -1.31640625, 0.0023555755615234375, -2.388671875, -0.229736328125, 1.1484375, 0.97265625, -0.58203125, -0.5400390625, -0.197021484375, 2.115234375, 0.207275390625, 1.421875, -0.92919921875, 1.669921875, -0.59765625, -1.2119140625, -1.4599609375, 0.28857421875, 0.076416015625, -1.2763671875, 0.92041015625, -0.146484375, 0.3037109375, -0.90771484375, -2.556640625, -0.904296875, 0.336181640625, 0.30517578125, 1.0390625, -0.7568359375, -1.537109375, -0.76806640625, 1.2900390625, -0.1907958984375, -0.74462890625, 0.0487060546875, 0.498779296875, 0.0220794677734375, -0.54150390625, -0.168701171875, -0.64892578125, -0.369140625, -1.1025390625, 0.58935546875, -1.2080078125, 0.20849609375, -1.2529296875, -0.37060546875, -3.263671875, -0.72314453125, -1.5712890625, 0.75830078125, -1.5703125, -0.113525390625, 1.619140625, -0.6611328125, 2.11328125, -0.72412109375, -0.41650390625, 0.94677734375, 0.0904541015625, -0.52001953125, 0.900390625, -0.36669921875, -0.2403564453125, 1.4912109375, 0.411376953125, -1.056640625, 0.947265625, -0.9375, -1.01171875, 0.5146484375, 0.1180419921875, -1.1884765625, 1.7216796875, -0.74560546875, -0.302978515625, -1.6787109375, 0.70166015625, -0.191650390625, -0.040283203125, 0.4228515625, 0.161865234375, 0.52587890625, -1.2412109375, 0.98388671875, 0.71044921875, -0.261962890625, -1.0361328125, 0.1873779296875, -0.0450439453125, 1.36328125, 0.01116180419921875, 0.267578125, 0.0004508495330810547, -0.39794921875, 0.338134765625, -2.07421875, 1.01171875, -1.6328125, -0.6630859375, -0.317626953125, -0.74755859375, -1.8017578125, 0.68505859375, -1.46484375, 1.6337890625, -1.328125, 0.152587890625, -0.71923828125, 0.3427734375, -1.4287109375, -0.312744140625, 1.603515625, -0.6982421875, 0.187255859375, 1.017578125, 1.01171875, -1.3134765625, 0.22216796875, -1.20703125, -0.69482421875, -0.488037109375, -0.2254638671875, -0.763671875, 0.974609375, -2.154296875, 0.6884765625, -0.322998046875, 0.91552734375, -0.6396484375, -2.4375, -0.52685546875, 1.4375, -0.83984375, 0.0014066696166992188, -0.327880859375, 2.22265625, 0.59130859375, -0.1529541015625, 0.058197021484375, -1.2255859375, -0.1905517578125, 1.443359375, 0.1846923828125, 0.77978515625, 0.0819091796875, -0.30615234375, 1.1572265625, -2.763671875, 0.509765625, -1.2802734375, 0.5546875, -1.1552734375, 1.1337890625, -0.92333984375, 0.1466064453125, 1.52734375, 0.595703125, 0.5400390625, -0.56298828125, 0.416259765625, 0.57275390625, 0.376708984375, -0.263671875, 0.424072265625, 0.56591796875, -1.1611328125, -1.513671875, 0.3876953125, 0.421630859375, 0.017852783203125, 0.232666015625, -0.427978515625, 0.65966796875, 1.1943359375, -1.7578125, 0.07525634765625, -1.2353515625, -0.2900390625, -0.81494140625, 0.923828125, 0.65185546875, -0.93212890625, -0.58056640625, -0.355712890625, 0.95703125, 0.263671875, 1.0078125, -0.8017578125, 1.09375, 0.88818359375, -0.1131591796875, 1.65625, -0.3173828125, 0.140380859375, 0.53076171875, -0.87890625, -0.78515625, 0.36083984375, 0.02655029296875, 0.8857421875, -0.2154541015625, 1.7392578125, -0.1126708984375, 0.267333984375, 0.3671875, -1.630859375, -2.208984375, -1.3330078125, 0.479248046875, 1.2666015625, 0.050933837890625, -0.54736328125, 0.10687255859375, 0.1978759765625, -0.2430419921875, 0.455810546875, 0.060028076171875, 2.216796875, 0.71240234375, 1.1943359375, 0.1666259765625, 1.09765625, 1.5185546875, -1.0283203125, 0.47705078125, 0.474853515625, -0.2381591796875, 0.9111328125, -1.0048828125, -0.669921875, -0.269287109375, -0.395751953125, 0.03936767578125, -1.1044921875, 1.0439453125, -0.290283203125, 0.560546875, -0.982421875, -1.4970703125, -0.385498046875, -0.43408203125, 1.1943359375, 0.52001953125, -0.279296875, -0.1412353515625, 0.1956787109375, 1.095703125, 0.46484375, 1.248046875, 1.005859375, 0.4443359375, -0.84033203125, -1.9072265625, 0.66943359375, 1.71875, 0.36669921875, -0.470458984375, 0.8564453125, 0.52734375, -1.33203125, 0.1676025390625, -1.44140625, -0.07135009765625, -0.263671875, 0.54638671875, 0.90869140625, -1.3935546875, -1.2578125, 0.521484375, -0.78369140625, -1.212890625, -1.0498046875, 0.6337890625, 0.65478515625, -0.0059356689453125, -1.43359375, -0.00780487060546875, 1.486328125, 0.87890625, 1.1298828125, -0.72314453125, -0.93408203125, 0.293212890625, -0.90283203125, 0.408447265625, 0.365478515625, -0.285400390625, -0.87353515625, -1.13671875, -0.28955078125, -0.468994140625, -0.72314453125, 1.22265625, 0.94580078125, 0.9755859375, 0.37060546875, -0.02838134765625, 0.2568359375, 0.5244140625, -0.1650390625, 0.2578125, -0.129638671875, 1.9140625, -0.787109375, -0.26953125, 0.77685546875, 1.9638671875, -0.498291015625, 1.5546875, -1.2158203125, -0.88525390625, -0.6064453125, 1.0546875, -1.3193359375, -1.9375, -0.4345703125, 0.78857421875, -0.7265625, 0.81591796875, 0.1529541015625, 1.841796875, 1.123046875, 1.0302734375, -1.9521484375, -0.464599609375, -1.220703125, 1.666015625, 0.66064453125, -1.5068359375, -0.439453125, 0.234619140625, -0.042144775390625, -0.2056884765625, 1.1025390625, -1.3173828125, 1.5576171875, -1.1650390625, -0.89013671875, 1.216796875, 0.1783447265625, -0.66357421875, -0.0811767578125, -0.0474853515625, 0.52978515625, -0.6591796875, 1.3330078125, -0.361083984375, -0.93408203125, 0.232666015625, -1.072265625, -0.0178375244140625, 0.94677734375, 0.49365234375, 1.6083984375, 1.388671875, 0.13720703125, 0.359375, -0.9287109375, -0.15771484375, -1.09765625, -0.51318359375, 0.0887451171875, -0.68212890625, 0.0029430389404296875, -1.802734375, -0.202392578125, -1.484375, 0.52880859375, -0.93408203125, 1.359375, -0.6796875, 1.7001953125, 1.8466796875, 0.6259765625, 0.420166015625, -0.400390625, -1.1875, 0.363525390625, 0.16357421875, -0.95947265625, -1.1357421875, -0.9375, 0.54638671875, -2.150390625, -0.54931640625, -1.1162109375, 1.6162109375, 1.2626953125, 0.6376953125, -0.356689453125, 0.9072265625, -0.60888671875, 0.5859375, -0.33447265625, -0.53173828125, 0.43603515625, 0.382080078125, 0.334228515625, 0.151611328125, -0.378662109375, 1.3427734375, 1.138671875, -0.291259765625, -0.246826171875, -0.85693359375, 2.763671875, 0.74365234375, 0.265380859375, -1.2568359375, 0.5576171875, 0.1798095703125, 0.80078125, -0.322998046875, 0.92236328125, -1.3134765625, 1.005859375, -1.490234375, 0.634765625, 2.021484375, -1.05859375, -1.0546875, -0.0689697265625, 1.1728515625, 1.3203125, 0.80810546875, 0.016937255859375, -0.176513671875, -0.0217437744140625, 0.7041015625, -0.023956298828125, 0.35009765625, 0.191162109375, 1.736328125, 0.927734375, -1.5390625, -0.32861328125, 2.826171875, -2.607421875, -1.857421875, 0.6279296875, 1.5224609375, -1.890625, 1.6611328125, 1.3095703125, -2.4296875, 0.53173828125, 0.8115234375, 2.890625, 0.38720703125, 0.8984375, -0.94189453125, -0.8232421875, 0.5029296875, 0.1329345703125, 0.340087890625, 0.48583984375, -0.59716796875, 0.302734375, -0.73583984375, 0.0469970703125, 0.56201171875, -0.2188720703125, -0.175048828125, -0.471923828125, 0.78271484375, -0.6318359375, 0.59326171875, -1.6630859375, 1.703125, -2.09765625, 0.51513671875, 1.0068359375, -1.333984375, -1.3447265625, 1.1181640625, -1.6650390625, 0.94482421875, -1.4482421875, 0.5087890625, -0.09857177734375, -0.52197265625, 1.4404296875, 0.611328125, -0.0171661376953125, 1.4775390625, 1.236328125, -0.369873046875, 0.6982421875, -0.287109375, 0.17431640625, -1.2158203125, 0.06317138671875, 0.2193603515625, -0.378662109375, -1.2529296875, 2.00390625, 0.5595703125, -0.78564453125, 0.432861328125, -2.466796875, -0.453369140625, -1.525390625, -0.51953125, -0.7470703125, -0.59130859375, 1.0478515625, -2.6796875, 0.1416015625, -0.724609375, -1.71875, -0.493408203125, -1.517578125, -0.3837890625, 1.9365234375, -1.326171875, -0.450439453125, -0.13623046875, 1.29296875, 0.27685546875, 0.88232421875, 0.501953125, 0.64013671875, -1.962890625, -0.62060546875, -0.2418212890625, -0.44482421875, -0.79150390625, 1.314453125, 0.315185546875, -1.5087890625, 2.3359375, 0.310302734375, -0.390869140625, 0.8427734375, 0.47802734375, -0.025238037109375, 1.2158203125, 0.154296875, -0.039703369140625, -0.3115234375, -0.474609375, 0.52587890625, 0.1407470703125, -0.19287109375, 0.65771484375, -0.52490234375, -0.381591796875, -1.142578125, 0.560546875, -0.2098388671875, -0.6103515625, -0.274658203125, -0.90185546875, -0.151611328125, 0.57275390625, -0.05853271484375, -1.0205078125, -0.187744140625, -0.44189453125, -0.2203369140625, -0.481689453125, -0.033538818359375, -0.6044921875, -0.34423828125, 1.7880859375, -0.75390625, 0.89453125, -0.4140625, -0.95068359375, 2.87109375, 1.564453125, -0.00927734375, -1.66796875, 0.7041015625, 0.453125, -0.66455078125, -1.248046875, -0.99072265625, -0.38525390625, 0.462646484375, -1.099609375, 0.1458740234375, 1.24609375, 2.107421875, -1.0498046875, 0.304931640625, 0.29638671875, 0.23876953125, 1.63671875, -2.1015625, 0.036041259765625, 1.4599609375, -0.0186004638671875, -0.431884765625, 1.0146484375, 0.005584716796875, -2.10546875, 1.0791015625, -0.44873046875, -0.98193359375, 0.210693359375, 0.80517578125, 1.244140625, 0.30029296875, 0.7509765625, -0.2257080078125, -0.29833984375, -0.3544921875, -1.4638671875, 1.134765625, -1.095703125, 0.0677490234375, 0.8369140625, 0.308837890625, -0.87060546875, -1.8896484375, 2.751953125, 0.6650390625, -0.080810546875, 0.736328125, 0.7734375, -0.63623046875, -1.541015625, -0.127685546875, -0.95068359375, 0.921875, 0.10870361328125, -0.2298583984375, 0.2218017578125, -0.693359375, -1.287109375, 0.24267578125, 1.740234375, -0.998046875, 0.58837890625, -0.81689453125, 1.5673828125, 0.30517578125, -0.3896484375, 0.94970703125, -0.412353515625, -0.2413330078125, 0.381591796875, -0.1995849609375, -0.31640625, -0.5595703125, 0.3310546875, -0.2078857421875, -0.552734375, -0.1851806640625, -0.5693359375, 0.036834716796875, 0.26806640625, 0.047454833984375, -0.049530029296875, 0.2457275390625, 1.982421875, 0.69970703125, -1.228515625, -0.1678466796875, -0.316650390625, 0.214111328125, -0.0301513671875, 0.1610107421875, 0.2100830078125, -1.3447265625, 1.3212890625, 1.0517578125, -1.287109375, -0.192138671875, 1.0478515625, 1.2783203125, 0.1373291015625, 0.112548828125, -0.329345703125, 0.3759765625, -0.60791015625, 1.09375, 0.0750732421875, 0.26513671875, 1.6279296875, -1.8935546875, 0.57666015625, -0.2386474609375, 0.203857421875, -0.55908203125, 0.5458984375, -0.52099609375, -0.5068359375, -0.8291015625, -0.958984375, -0.2364501953125, 0.10772705078125, -0.67236328125, 0.0640869140625, 1.3125, -0.7236328125, 1.7587890625, 1.5888671875, -1.1669921875, 0.052978515625, 1.208984375, -1.2216796875, -0.71435546875, -1.5751953125, 0.340576171875, -0.77734375, -0.99169921875, -0.56103515625, -0.32861328125, 0.53271484375, 0.12274169921875, -2.087890625, 0.87109375, 1.4384765625, -2.626953125, 0.255859375, -0.272216796875, 1.1884765625, 0.779296875, 0.6123046875, -1.1767578125, -2.9765625, 0.1962890625, 0.60107421875, -1.6884765625, 1.65234375, -0.64599609375, -0.00798797607421875, 0.322998046875, 0.266845703125, 0.90771484375, -0.456787109375, 0.4072265625, -0.6240234375, 0.1064453125, 0.260986328125, 0.5732421875, 0.373779296875, 1.01171875, -0.28759765625, -0.6171875, -0.044403076171875, 1.7626953125, 1.84375, -1.55859375, 1.3125, 0.9306640625, -0.469970703125, -0.10406494140625, -1.0283203125, -0.86474609375, -0.7041015625, -0.2607421875, 0.505859375, 1.0595703125, -1.400390625, -0.59716796875, -1.095703125, 0.8544921875, -0.4423828125, 0.6806640625, -0.290283203125, -0.029296875, -0.2291259765625, -0.481689453125, -0.85205078125, 0.058807373046875, -1.70703125, 0.0992431640625, 0.1439208984375, 1.9609375, 0.9951171875, -0.0684814453125, -0.467041015625, 1.4150390625, -1.5654296875, 2.314453125, 1.39453125, -0.98779296875, -0.61376953125, -0.290771484375, -0.12298583984375, -2.521484375, 0.3408203125, -0.353271484375, -0.58642578125, 0.953125, 0.0007319450378417969, -0.1575927734375, 1.625, 1.0078125, -0.333984375, 1.2119140625, 0.84716796875, -1.578125, -0.828125, -0.11822509765625, 0.0022373199462890625, 1.037109375, 0.7373046875, 0.23291015625, 0.06292724609375, 1.0791015625, 0.01297760009765625, -0.25439453125, -0.45166015625, 1.1357421875, 1.30078125, -0.218994140625, -1.82421875, -0.55029296875, 0.295654296875, 0.9208984375, 1.6171875, -0.094970703125, 0.337646484375, 0.436279296875, 0.96875, -2.005859375, 0.11798095703125, -0.0005040168762207031, -0.80322265625, -0.546875, -0.3701171875, 0.08392333984375, 0.2010498046875, 0.0689697265625, 1.4189453125, -1.6337890625, -1.0498046875, -0.73486328125, -1.0166015625, -1.123046875, -0.245849609375, -0.332763671875, 1.1650390625, -0.480224609375, -1.2333984375, -0.256103515625, -0.95361328125, 1.0703125, 2.31640625, 0.242919921875, 1.5966796875, -0.0938720703125, -0.8505859375, 0.5634765625, -0.14111328125, 0.2181396484375, -1.8779296875, 1.015625, -1.322265625, 2.916015625, 1.04296875, -0.394287109375, 0.82275390625, 0.2105712890625, -1.2275390625, 0.98095703125, -0.0963134765625, 0.8740234375, -1.876953125, -1.2431640625, 0.43212890625, -0.27587890625, 0.343994140625, -0.351318359375, 0.01090240478515625, 0.0005855560302734375, 2.376953125, 0.482421875, -0.33544921875, 1.0693359375, -0.3876953125, -0.62548828125, -1.4814453125, -0.2474365234375, 1.2783203125, -0.712890625, -1.4580078125, -0.1558837890625, 0.6376953125, -0.939453125, 0.93359375, 0.75439453125, 1.1220703125, -0.53662109375, 0.10943603515625, -0.9384765625, 1.14453125, -1.3818359375, -1.10546875, 0.414794921875, 0.84912109375, -2.125, -0.66796875, 0.08856201171875, -0.2147216796875, 0.79931640625, 1.205078125, 0.256591796875, 1.25, -0.99365234375, 1.09765625, 0.322021484375, 0.51025390625, 0.66845703125, -1.0380859375, 0.95068359375, 0.3505859375, 1.5537109375, 0.47509765625, -0.533203125, -1.1884765625, -0.95703125, -0.2344970703125, 0.712890625, -0.6376953125, 1.2607421875, 1.6005859375, 0.26123046875, -0.84521484375, -0.36669921875, -0.09002685546875, 0.0007700920104980469, -0.05511474609375, 2.1953125, -1.0087890625, -0.90869140625, -2.75390625, -0.060089111328125, -0.87646484375, 0.56201171875, -0.7177734375, 0.00307464599609375, 0.045257568359375, 0.249267578125, 1.583984375, 1.0849609375, 0.48681640625, 1.326171875, 2.232421875, -1.8076171875, -2.51171875, 0.9775390625, -0.26123046875, 0.66064453125, 1.5458984375, -1.353515625, 0.08563232421875, -1.9287109375, -0.255859375, -0.21142578125, -1.244140625, 0.392578125, 0.415771484375, 0.49365234375, 2.56640625, 0.72607421875, -1.5625, 0.1563720703125, 0.669921875, 0.27978515625, 0.392333984375, -1.375, -1.1123046875, 1.0654296875, -0.59033203125, 1.185546875, 2.076171875, 1.1435546875, -0.249267578125, -0.41357421875, 0.1373291015625, -0.97802734375, 0.2225341796875, -0.45556640625, -0.107421875, 0.0693359375, -0.9765625, -0.34375, -1.15234375, -0.06829833984375, -0.409423828125, 0.72998046875, 0.413330078125, 0.185791015625, 0.77685546875, 1.9140625, -1.4208984375, 0.8876953125, 0.327392578125, 1.2724609375, -0.069580078125, -1.306640625, -1.546875, -0.057464599609375, 1.294921875, 0.387939453125, -2.1171875, 0.71826171875, -0.68896484375, 0.06756591796875, -0.21533203125, 0.2177734375, -0.2939453125, 0.70849609375, -1.3876953125, 0.576171875, 1.072265625, 0.346923828125, 1.4638671875, -0.55322265625, 0.59375, -0.1925048828125, 0.74365234375, -2.314453125, 1.0126953125, -1.3212890625, -0.384521484375, 1.4951171875, -0.669921875, -0.57763671875, 0.0684814453125, -2.125, 0.8564453125, -1.2216796875, 0.5341796875, -0.06341552734375, 0.2939453125, 0.353515625, -1.4716796875, -0.78955078125, 0.495849609375, 0.44921875, 0.7177734375, -1.6396484375, -0.74658203125, -0.69970703125, 0.57763671875, 1.6982421875, 0.56591796875, -1.6435546875, -2.294921875, -0.30224609375, 2.046875, -0.9248046875, -1.2373046875, -0.8125, -0.496826171875, 0.296142578125, 1.0673828125, -1.0927734375, -1.197265625, 2.16015625, -0.62353515625, 0.3779296875, -0.56591796875, 0.455810546875, 2.25390625, -0.748046875, 0.34765625, -1.099609375, -0.3173828125, -0.51611328125, -0.468505859375, -0.0762939453125, 0.5810546875, 0.67626953125, 0.6240234375, -0.239501953125, -0.87109375, -0.556640625, 0.344970703125, 0.304443359375, 1.45703125, -0.263427734375, 0.365478515625, -0.51708984375, -1.01171875, 0.82080078125, -0.6455078125, -0.01154327392578125, -0.310302734375, -1.1416015625, -0.1143798828125, -2.181640625, -0.08929443359375, 1.740234375, 1.580078125, 1.0380859375, 0.87744140625, 0.420166015625, -0.6123046875, -0.14404296875, -0.1541748046875, 1.404296875, 0.416259765625, 1.26171875, -0.84765625, 0.77734375, -0.213623046875, -2.26953125, -1.3349609375, 0.697265625, -0.63525390625, 1.12890625, 0.379150390625, 0.95263671875, -1.982421875, -0.6845703125, 0.39697265625, -0.8564453125, 0.9609375, -2.296875, 1.3017578125, -1.4970703125, 0.380859375, -0.55810546875, 1.0283203125, 0.1175537109375, -0.79931640625, -0.267333984375, 1.5029296875, -0.60302734375, -1.6298828125, 0.1815185546875, -0.264404296875, 0.4169921875, -0.79541015625, 0.87255859375, 0.290771484375, 0.284912109375, 1.78125, -0.43359375, 0.98046875, 1.099609375, -1.9453125, 2.51171875, 0.1622314453125, 0.056243896484375, 1.462890625, -1.3671875, 0.52490234375, -0.80810546875, -0.228271484375, 1.560546875, 0.2369384765625, 1.140625, -0.572265625, -1.478515625, 1.6650390625, 0.6552734375, 0.50244140625, -0.56640625, 0.287841796875, 1.0205078125, -0.6201171875, 0.2291259765625, 0.8779296875, -1.3818359375, 0.47216796875, -0.036376953125, -1.578125, -0.72119140625, 0.279296875, 0.371826171875, 0.7978515625, 1.6376953125, -0.5244140625, -0.283935546875, 0.916015625, -0.544921875, 0.76318359375, -0.98046875, -1.6865234375, -0.93408203125, -1.17578125, 3.169921875, -0.41259765625, -0.48828125, 0.73828125, -0.1097412109375, -0.27783203125, -0.1295166015625, -0.7041015625, 1.03125, 0.9091796875, -0.1800537109375, -0.0789794921875, 0.2108154296875, 1.00390625, -0.03582763671875, -0.572265625, -0.042510986328125, -1.400390625, -0.93310546875, 0.1343994140625, -0.6376953125, 1.326171875, -1.4404296875, 0.2099609375, 0.865234375, -1.0849609375, 0.84423828125, -0.62548828125, 0.60986328125, -0.361083984375, -0.8701171875, -1.7890625, -0.421630859375, -0.78515625, -0.1260986328125, -0.34130859375, 0.40966796875, -1.7900390625, -0.2208251953125, -0.59619140625, 0.7587890625, -0.74365234375, 1.4033203125, -0.032379150390625, 0.04693603515625, -0.1268310546875, 0.0109710693359375, 0.59228515625, -1.6748046875, 1.3984375, 0.9443359375, -0.44580078125, 1.0703125, -0.5576171875, -1.1953125, -0.99169921875, 0.1649169921875, -1.2109375, -0.2032470703125, -0.185546875, -0.7646484375, 0.58984375, 2.009765625, -0.043121337890625, -1.2158203125, 0.470703125, -0.1419677734375, -0.255859375, -0.292724609375, -0.64892578125, 0.55810546875, -0.394775390625, 0.276611328125, -0.0128021240234375, 0.8359375, 1.4443359375, 0.380126953125, 0.314208984375, -0.888671875, -1.130859375, 1.072265625, 0.2254638671875, -0.4375, 0.1298828125, 0.271484375, 0.285888671875, 0.417236328125, -0.40771484375, -0.9638671875, 0.62255859375, 0.51806640625, 1.2978515625, 0.78173828125, -1.5576171875, 0.1790771484375, -0.3115234375, 1.0185546875, -0.9814453125, -0.66845703125, 0.5302734375, -1.8681640625, 0.2259521484375, -0.331787109375, -1.3427734375, 0.87939453125, 1.2919921875, 1.2421875, 1.0771484375, -0.10614013671875, 0.438232421875, -0.178955078125, 0.056243896484375, 0.7587890625, -0.00814056396484375, 0.6767578125, -1.125, 0.4228515625, -0.181884765625, -1.0302734375, 0.40576171875, 0.0218505859375, 0.6328125, -0.4384765625, -0.1923828125, 0.2447509765625, 1.0380859375, -0.310302734375, 1.0087890625, 0.4814453125, 1.615234375, 0.96923828125, 0.80126953125, 1.185546875, 0.12457275390625, 1.0498046875, 0.248046875, -0.51025390625, -1.72265625, 0.97021484375, 0.5869140625, -2.724609375, 0.939453125, -0.5380859375, 0.2137451171875, 1.025390625, -0.017425537109375, -0.460693359375, 0.7099609375, -0.318359375, 0.15087890625, 2.126953125, 0.4052734375, 0.245361328125, -1.787109375, 0.062469482421875, -1.0478515625, -0.97900390625, -1.025390625, -0.8427734375, -0.791015625, -0.66650390625, 0.3974609375, -1.19921875, -0.525390625, -0.83203125, -0.4892578125, 0.351318359375, -1.3798828125, 1.80078125, 0.079833984375, -1.5263671875, -1.451171875, -0.5126953125, 0.003147125244140625, 0.350341796875, -0.8505859375, 0.92529296875, 1.3291015625, 0.197021484375, 0.1942138671875, -1.67578125, 0.58203125, 1.1220703125, 0.75244140625, 0.65966796875, 0.0247802734375, 0.11163330078125, -0.42919921875, -0.5947265625, -1.12890625, -0.247314453125, 0.2098388671875, -0.8447265625, 1.533203125, 0.055755615234375, -0.410888671875, 1.017578125, 0.07598876953125, -0.001789093017578125, -2.59375, -0.344482421875, -1.2333984375, -1.638671875, 1.765625, -0.67431640625, 0.420654296875, 1.501953125, -1.025390625, 0.492431640625, -1.15234375, 0.7265625, 0.32373046875, 1.896484375, 1.44921875, 0.300537109375, -0.56298828125, -0.5048828125, 0.08856201171875, 0.322998046875, 0.263671875, 0.0618896484375, 0.865234375, 0.56640625, -0.5888671875, 0.99951171875, 0.255859375, 0.62939453125, -0.07305908203125, 2.376953125, -0.78076171875, 1.15234375, 2.142578125, -0.7998046875, 0.0966796875, -0.607421875, -1.1650390625, -1.486328125, -0.9033203125, 1.068359375, -0.93359375, 0.08685302734375, -0.8330078125, 0.1798095703125, 0.54931640625, -0.09765625, 0.0709228515625, -0.1624755859375, 0.603515625, -1.3359375, 0.34716796875, -0.53515625, 2.00390625, 0.1898193359375, 1.7236328125, 0.382568359375, -0.37548828125, 0.07763671875, 1.572265625, 0.60888671875, -0.95654296875, -1.2236328125, -0.6201171875, 2.28125, -0.2398681640625, -2.0234375, -0.398681640625, 0.12469482421875, -1.0888671875, -1.6337890625, 0.09716796875, 0.7451171875, 1.5693359375, -0.931640625, -1.396484375, -1.30859375, 1.0302734375, 0.46630859375, -0.51904296875, 1.0087890625, -0.572265625, 1.3369140625, 0.525390625, -0.481201171875, 1.1640625, -0.444580078125, -1.8974609375, -0.85205078125, -1.3525390625, -0.0806884765625, -0.001697540283203125, 1.447265625, 0.2861328125, -1.638671875, -0.322509765625, -0.79736328125, -0.4482421875, -0.1624755859375, -0.2130126953125, 0.307861328125, -1.619140625, 0.697265625, -1.353515625, 0.2481689453125, -1.5634765625, 1.2509765625, -1.8017578125, 1.654296875, -0.68115234375, 0.43310546875, 0.8154296875, -0.48583984375, -1.1884765625, 1.0068359375, -0.366455078125, 0.48779296875, -0.54443359375, 1.4599609375, -0.2135009765625, 1.55859375, -0.83935546875, 1.0390625, 1.0029296875, 0.8427734375, 1.275390625, 1.1650390625, 1.537109375, 0.36474609375, -2.2421875, -0.284912109375, 0.12188720703125, 1.224609375, -0.814453125, 0.19189453125, 1.7001953125, 0.271484375, 0.62255859375, -0.48681640625, 0.89208984375, -0.66455078125, 0.29052734375, 1.2734375, -1.4072265625, 0.047210693359375, 0.71533203125, -0.114501953125, 0.8876953125, -0.92529296875, -1.341796875, 0.2352294921875, -0.41064453125, 0.41015625, -1.0986328125, -0.1563720703125, -0.44873046875, 0.19482421875, -0.254638671875, 0.263916015625, -0.51953125, 0.84619140625, -0.141845703125, -1.7734375, -2.240234375, 0.433837890625, -0.8056640625, 0.085693359375, -1.5048828125, -0.69580078125, -0.9619140625, 2.666015625, 1.4462890625, 0.386474609375, 2.310546875, -0.61279296875, -0.01087188720703125, -1.2041015625, -0.384033203125, 0.73828125, -1.2958984375, 0.267333984375, -0.01265716552734375, 0.1146240234375, 0.1336669921875, 1.1845703125, -1.2197265625, 0.81787109375, 1.419921875, -0.250732421875, 0.117919921875, 1.0478515625, -1.0166015625, 2.291015625, 1.4931640625, 1.455078125, 0.96435546875, 0.374267578125, 0.6142578125, 1.216796875, 0.346923828125, 1.5986328125, -0.22216796875, -0.0628662109375, 2.23046875, -2.080078125, -0.71337890625, 0.92431640625, 0.14404296875, 0.68798828125, 0.640625, 0.96826171875, 1.1640625, 1.583984375, 1.7041015625, -1.5361328125, 0.5068359375, 1.4111328125, 0.0007872581481933594, -1.9755859375, -0.169921875, -2.38671875, -0.386962890625, 1.3828125, 0.86962890625, 1.7578125, 0.76611328125, 0.275634765625, 0.142822265625, -1.6083984375, -0.6201171875, -0.77880859375, 0.35302734375, -0.91748046875, 0.71240234375, 1.59765625, 1.65625, -1.1044921875, -1.2080078125, -0.94677734375, 1.4892578125, 1.2060546875, -0.74365234375, 0.65869140625, -0.11676025390625, 0.54150390625, -1.12890625, 1.625, -1.216796875, -1.37890625, 0.11334228515625, -0.65576171875, -0.64404296875, -0.67626953125, -0.38037109375, 1.072265625, -0.5712890625, -1.25390625, 0.81201171875, -0.60498046875, -0.334228515625, 1.0849609375, -0.31884765625, 0.203369140625, -1.0107421875, 2.03125, -0.253662109375, -0.6513671875, -0.0018634796142578125, 0.50146484375, -0.135009765625, 1.0087890625, 1.0, -0.841796875, -0.1668701171875, 1.1416015625, 0.1961669921875, 1.2978515625, 0.70068359375, -2.142578125, -1.125, -0.372802734375, -0.1995849609375, 0.609375, -0.6845703125, -0.315185546875, 0.17431640625, -0.361083984375, -0.425537109375, 0.21630859375, -1.42578125, 0.9423828125, 0.45849609375, -0.67919921875, 0.66357421875, 0.47509765625, 0.1729736328125, 0.0806884765625, 1.6787109375, -0.41845703125, 0.094970703125, -1.0107421875, 0.6787109375, -0.1300048828125, 0.1444091796875, -0.69140625, -0.22412109375, -0.0496826171875, 0.392578125, -1.2412109375, 0.62841796875, 0.311279296875, 0.23291015625, 0.78955078125, 0.544921875, 0.179931640625, 0.044189453125, -2.33203125, -1.8330078125, 2.72265625, 0.74658203125, 0.1807861328125, 1.4384765625, -0.36474609375, -0.472412109375, 0.83642578125, -1.0224609375, 0.10382080078125, -0.413818359375, -0.092529296875, 0.509765625, -0.52783203125, 1.482421875, 0.544921875, -0.330810546875, -0.290771484375, 1.6484375, 0.2186279296875, -0.5712890625, -0.56884765625, -0.333984375, -0.231201171875, 0.083251953125, -1.087890625, 1.13671875, -0.0245208740234375, 0.192138671875, 0.10479736328125, -0.424560546875, -0.390625, -1.556640625, 0.974609375, 1.7685546875, -0.8896484375, -0.38037109375, -1.01953125, 0.340576171875, -0.84521484375, 0.9482421875, -1.3349609375, -1.595703125, 0.88720703125, -0.90673828125, -0.73388671875, -0.8466796875, 0.5751953125, 0.63427734375, 0.26123046875, 1.6611328125, 0.45703125, -1.1357421875, -0.50732421875, -0.69189453125, 0.468994140625, 0.6865234375, 1.5498046875, 0.59130859375, 1.7431640625, -0.73046875, 0.88818359375, 0.451904296875, -1.84765625, -0.5859375, 0.06060791015625, 0.87060546875, -1.1708984375, -1.0048828125, -0.078857421875, 0.74755859375, -0.70947265625, 0.8603515625, -1.2724609375, -0.3857421875, -0.42578125, -1.0859375, 1.400390625, 0.77978515625, -0.2296142578125, 0.06353759765625, -0.34716796875, -0.03466796875, -0.34130859375, 0.381591796875, 0.92919921875, 0.7158203125, 0.474853515625, 2.060546875, 1.4638671875, 2.37109375, -1.189453125, 0.21337890625, -0.5556640625, -0.484375, -0.8837890625, 1.2431640625, 0.6181640625, 0.2120361328125, 1.291015625, -2.72265625, -0.465087890625, -0.81201171875, 2.12890625, -0.55126953125, 0.67236328125, -1.328125, 2.17578125, -0.31103515625, -0.72314453125, 0.050048828125, 1.13671875, 0.1800537109375, 0.413330078125, -0.4248046875, 0.4619140625, 0.80126953125, -1.9375, -1.0009765625, 1.2919921875, -2.0546875, -0.327392578125, 0.55322265625, -0.7822265625, -1.06640625, -0.445068359375, 0.3154296875, -1.0107421875, 0.88037109375, -0.513671875, 1.4775390625, 0.794921875, 0.375732421875, 0.197021484375, -0.1300048828125, 0.89404296875, 0.11627197265625, -1.59375, -0.04510498046875, 0.1717529296875, -0.199951171875, 0.360107421875, 0.87451171875, 0.279541015625, -0.330078125, 0.55029296875, -0.00972747802734375, 0.36767578125, -1.7255859375, -0.385986328125, -1.2939453125, 1.1845703125, 1.0556640625, 1.3251953125, -0.476318359375, 0.09906005859375, -0.144775390625, -0.254150390625, -0.73681640625, 0.00286865234375, -0.2471923828125, -0.208740234375, 0.01430511474609375, 0.25390625, 1.8427734375, 1.0078125, 0.8505859375, -0.138671875, -0.56103515625, 0.69921875, 0.50390625, -1.7021484375, 0.07940673828125, -0.4189453125, 0.155517578125, -0.92041015625, -0.410400390625, -1.22265625, 0.779296875, -1.7197265625, 1.9423828125, 0.3974609375, 1.8203125, -1.4189453125, 1.02734375, -0.464111328125, -0.7744140625, -0.75244140625, -0.71142578125, -0.06744384765625, -0.88916015625, 0.70751953125, -0.3525390625, 0.22998046875, 1.3427734375, -0.65380859375, 1.4365234375, 0.77978515625, 0.53515625, -0.275390625, -1.09375, 0.73046875, 0.80322265625, 0.322265625, 0.52685546875, 0.32080078125, 0.634765625, 0.34033203125, 1.8720703125, -1.0673828125, -0.60791015625, -1.7890625, -0.25927734375, 0.56591796875, 0.08953857421875, -0.28759765625, -0.359375, -0.673828125, 1.763671875, -1.419921875, -0.6474609375, 1.2919921875, 0.7294921875, -1.17578125, -2.0546875, 2.0703125, -1.2900390625, -0.4404296875, -0.6064453125, -1.0634765625, -0.6689453125, -1.1826171875, -0.916015625, 1.1201171875, 1.5927734375, -0.1453857421875, 0.297607421875, -0.88427734375, -0.32763671875, -0.69677734375, -0.0980224609375, 0.326171875, 1.2177734375, 0.144775390625, 0.88671875, -1.0634765625, 1.7822265625, 1.470703125, -0.77978515625, -0.10455322265625, 1.44921875, 0.3037109375, -0.72607421875, -0.947265625, 0.9609375, -1.490234375, -1.310546875, -1.544921875, 1.3779296875, -0.927734375, 0.796875, 1.1865234375, -0.031585693359375, 2.23828125, 0.011077880859375, -0.8486328125, 2.109375, 1.3583984375, -0.00628662109375, -0.65234375, 1.25, 0.951171875, -1.93359375, 0.314453125, 0.363525390625, -0.275634765625, 0.59619140625, 0.375732421875, 1.763671875, -0.77783203125, -0.045379638671875, -1.6044921875, -0.9658203125, 1.53515625, -1.0009765625, -0.361572265625, -0.048797607421875, 1.74609375, -0.615234375, -0.54248046875, 0.459716796875, 0.10150146484375, -0.5771484375, -0.7705078125, -2.30859375, -0.058746337890625, -0.2242431640625, 0.77099609375, -0.50146484375, -0.255859375, -0.17041015625, -1.4248046875, 0.619140625, -0.101318359375, -0.309326171875, -0.79248046875, -1.71484375, 0.85302734375, -0.796875, -0.255859375, 0.61865234375, 1.724609375, -0.037384033203125, 0.73828125, 0.744140625, 1.0087890625, -1.14453125, -1.8798828125, -1.2548828125, -0.09442138671875, -0.60693359375, 0.53515625, -0.95654296875, 0.513671875, 0.80419921875, -1.515625, -0.1070556640625, -0.9140625, 1.2080078125, -1.28515625, 0.591796875, -1.4462890625, -0.00432586669921875, 0.97607421875, -0.463623046875, 0.2259521484375, 0.78125, -1.3369140625, 1.53515625, -0.41015625, -0.77880859375, -1.6103515625, 0.456787109375, 1.6962890625, -1.4189453125, -0.9033203125, -1.1611328125, 0.1580810546875, 2.259765625, -0.7958984375, 0.385986328125, 0.98291015625, -2.546875, 0.73046875, 1.0302734375, 0.461669921875, 0.328369140625, 0.59619140625, -0.3544921875, 0.18359375, -0.05731201171875, 0.043365478515625, 2.11328125, 0.17431640625, -0.258056640625, -2.494140625, 0.5537109375, 0.2064208984375, -0.0732421875, -0.06219482421875, 0.237060546875, -1.6083984375, -1.1611328125, -0.07379150390625, 0.284912109375, -0.1800537109375, -0.287109375, 0.8427734375, 0.43115234375, -0.1527099609375, -0.384765625, 0.9765625, 0.880859375, -0.194091796875, 0.1224365234375, -0.48486328125, -0.1712646484375, 0.75, -0.720703125, 0.580078125, -1.375, -0.56884765625, 0.1004638671875, -1.4697265625, -0.369384765625, 1.5263671875, 0.88671875, 0.6064453125, 1.0419921875, -1.0224609375, -0.509765625, -0.78955078125, 0.3251953125, -0.1175537109375, -0.77734375, 0.155517578125, -1.796875, 0.2037353515625, -0.34375, -0.220703125, 0.047088623046875, 0.2156982421875, 2.494140625, 0.634765625, -2.052734375, -1.5, -0.34619140625, -1.3115234375, 1.259765625, 1.5712890625, 0.1732177734375, -1.27734375, 0.57275390625, 0.93359375, -0.61083984375, 1.634765625, -0.60693359375, 1.90234375, -0.6611328125, 1.56640625, 0.66162109375, 0.474609375, 0.5625, 1.4609375, -1.74609375, -1.8994140625, 1.447265625, 1.2158203125, 1.345703125, -0.381591796875, -0.53369140625, 1.283203125, -0.3837890625, -0.464111328125, 0.71826171875, -0.57373046875, 0.1495361328125, 0.5703125, -0.32177734375, 0.460693359375, 0.365234375, -0.37744140625, 0.8759765625, 0.334228515625, -1.2548828125, 0.66796875, -0.458740234375, -1.591796875, 0.15380859375, -0.490234375, -1.3017578125, -0.0159149169921875, 0.75732421875, -2.849609375, -0.42724609375, 0.11614990234375, 0.329833984375, 0.77978515625, 1.4072265625, 0.4619140625, 0.315185546875, -2.701171875, 0.1162109375, -0.45166015625, -0.71826171875, 1.0400390625, -0.2091064453125, -0.495361328125, -0.32470703125, 0.203369140625, -0.27587890625, -1.8330078125, -0.265625, -0.415283203125, 1.6650390625, 0.72265625, -1.4736328125, 2.1875, 0.474609375, 2.0078125, 0.5205078125, 3.04296875, -0.487060546875, -0.274169921875, 0.90185546875, -0.32275390625, 1.4228515625, -0.43310546875, 0.041961669921875, 0.72607421875, -0.497314453125, 0.340087890625, 0.068603515625, 1.2939453125, -2.76171875, 0.0728759765625, -0.1385498046875, 0.407958984375, -0.35546875, 0.01483154296875, -0.865234375, -0.207275390625, -0.53369140625, 1.7275390625, -0.86083984375, -0.7802734375, 0.365478515625, 2.4765625, -0.99755859375, -0.736328125, -0.62890625, -0.439697265625, 0.5205078125, 0.88427734375, 0.59765625, 1.064453125, 0.48388671875, -0.7236328125, -0.36865234375, -1.3740234375, 0.431396484375, 0.9033203125, -0.3154296875, -1.3271484375, -0.3681640625, -0.7841796875, -0.59814453125, -0.9482421875, 2.201171875, 0.52734375, 2.072265625, 0.16845703125, -0.728515625, -0.1407470703125, -0.98291015625, -1.025390625, -0.8994140625, -0.82421875, 0.43603515625, -1.0576171875, 0.07232666015625, 0.8544921875, -1.0751953125, -0.468505859375, 0.160888671875, -1.052734375, 1.6318359375, -1.720703125, 1.328125, -1.021484375, -1.251953125, -0.806640625, -1.7705078125, -1.72265625, -2.068359375, 0.6064453125, -0.407470703125, 1.1923828125, 1.3125, -0.058746337890625, 0.04608154296875, 0.62451171875, 0.62646484375, -1.322265625, -0.8876953125, -0.50048828125, 0.306396484375, -0.7939453125, -0.1124267578125, -1.669921875, -0.189208984375, -1.9248046875, -0.09039306640625, -0.94873046875, 1.646484375, -1.7314453125, -1.001953125, -0.4892578125, 0.97509765625, -0.43994140625, -0.1416015625, 0.9248046875, -0.5810546875, -0.486083984375, 0.09320068359375, 0.8701171875, 2.02734375, 0.99658203125, 1.0078125, 1.8291015625, 1.140625, -0.388671875, -0.70458984375, 0.0292205810546875, -0.1275634765625, -0.07373046875, -0.3525390625, 0.324951171875, -2.2734375, -0.91015625, 0.93408203125, 0.8740234375, 0.5693359375, 0.5322265625, 1.056640625, -0.7880859375, -0.919921875, -0.72216796875, -1.1162109375, -1.3984375, -0.326416015625, -0.29345703125, -0.26171875, 0.6904296875, 0.374755859375, 0.399658203125, 1.9228515625, 1.0908203125, 1.3720703125, 0.1519775390625, -1.728515625, 0.483154296875, 0.284423828125, -1.8623046875, -0.6728515625, 0.41015625, 2.13671875, 0.337158203125, 0.019744873046875, -0.8212890625, 0.22412109375, -0.744140625, 0.0972900390625, -0.9580078125, -0.2037353515625, -1.2197265625, -1.2646484375, -0.87353515625, 0.85986328125, 0.06414794921875, 1.232421875, -1.2900390625, -0.994140625, 1.3935546875, 0.054107666015625, 0.1842041015625, 1.3447265625, 2.35546875, -0.184326171875, -0.1527099609375, -0.55078125, 0.53857421875, -1.81640625, -1.681640625, -0.77099609375, 1.7578125, -1.3212890625, -0.13623046875, -1.5576171875, 0.78662109375, -0.321533203125, 0.62353515625, 0.3623046875, -0.3759765625, -0.31005859375, 0.0926513671875, -0.83544921875, 1.482421875, -0.88427734375, -0.1590576171875, -1.482421875, 1.123046875, 0.88525390625, 0.72314453125, 1.0712890625, -0.4189453125, -1.4833984375, 2.3125, -1.544921875, -0.305419921875, -1.134765625, -0.97900390625, -0.82421875, 0.7705078125, 0.2093505859375, -0.5068359375, 0.02392578125, 1.4228515625, 0.57421875, -0.46142578125, 0.6455078125, 1.03125, 0.58740234375, 0.90966796875, 0.8642578125, -0.9375, 0.06695556640625, -2.04296875, -0.0006275177001953125, 0.173828125, -0.541015625, -1.125, -1.1279296875, 0.86376953125, 0.6494140625, 0.77001953125, 0.6806640625, -1.7080078125, -0.1910400390625, -0.962890625, -0.041778564453125, -1.37890625, 0.4853515625, 0.708984375, 0.58837890625, 0.460693359375, -1.0458984375, -0.4482421875, -1.4990234375, -1.10546875, -0.218017578125, 0.10089111328125, -1.4794921875, -0.1412353515625, 1.4013671875, 0.82275390625, 0.50732421875, 1.6376953125, 0.89794921875, 1.53125, -0.61474609375, 0.4716796875, -1.0439453125, 0.63330078125, -0.52734375, -0.07470703125, -1.052734375, -2.33984375, -0.0017347335815429688, -1.8232421875, -0.452392578125, 0.6572265625, -0.29541015625, 0.6474609375, -0.40576171875, -1.3349609375, 1.5869140625, 0.25830078125, -0.35888671875, 0.1962890625, -0.08270263671875, 1.6181640625, -0.278564453125, -1.0166015625, -0.37158203125, -0.87353515625, 0.490234375, 0.179931640625, 0.83056640625, 0.09796142578125, -1.5087890625, 0.47705078125, -0.427734375, 0.9013671875, -0.407958984375, -2.732421875, 1.2919921875, -0.41455078125, -0.6875, -0.833984375, 0.430908203125, 2.115234375, 0.58349609375, 1.458984375, 0.830078125, 0.260986328125, -0.0560302734375, -0.31494140625, 0.943359375, 0.2349853515625, 2.009765625, 0.6962890625, 0.045684814453125, 0.2447509765625, 1.125, 2.546875, 0.10064697265625, -1.162109375, -0.088623046875, -0.054962158203125, -0.916015625, 0.58544921875, -0.48681640625, -0.158203125, 0.26806640625, 1.48046875, -0.802734375, -0.03411865234375, -0.286376953125, -0.97119140625, -0.3740234375, -0.93603515625, 0.317626953125, -0.0748291015625, 0.419189453125, 0.93212890625, 1.6787109375, -0.41650390625, -0.4736328125, 2.123046875, -0.51513671875, -0.71728515625, -2.412109375, 0.9287109375, 1.796875, -0.359619140625, 0.1318359375, 0.12200927734375, -1.2470703125, 1.8251953125, -0.59423828125, 0.59814453125, -1.763671875, -0.18505859375, -1.197265625, -0.58056640625, 0.5068359375, 0.261474609375, -0.4091796875, -2.203125, 0.389404296875, -0.88671875, 0.68896484375, -0.0975341796875, 0.0909423828125, 0.53515625, -0.91650390625, -0.210693359375, -1.19140625, -0.1510009765625, -0.9462890625, -0.1256103515625, -0.1353759765625, -0.57177734375, 0.685546875, -0.9345703125, -0.58251953125, 0.67529296875, 0.435302734375, 0.335693359375, -0.92431640625, -1.1337890625, -0.024200439453125, 0.63916015625, 0.11468505859375, 0.71728515625, 1.2978515625, 0.005725860595703125, -2.3984375, -1.96875, 1.13671875, 0.5654296875, 0.1484375, 0.22607421875, -1.1220703125, 2.2890625, -0.79052734375, 0.77734375, -0.17529296875, 0.371826171875, -0.192626953125, -1.037109375, 1.02734375, 0.215576171875, -0.76611328125, -1.5712890625, -0.435791015625, 0.72314453125, 0.72705078125, -0.93115234375, 1.1279296875, 0.67724609375, 0.30419921875, -0.8232421875, 0.58447265625, -0.443359375, -0.9970703125, 1.46875, 0.1981201171875, 1.1962890625, -1.18359375, 0.060272216796875, 0.943359375, -1.1708984375, -0.323486328125, 1.0986328125, -1.1611328125, -0.70654296875, -1.2490234375, 1.0517578125, -0.5576171875, 0.316650390625, 1.91796875, -1.423828125, -2.18359375, -0.345947265625, -1.2373046875, 0.450439453125, -1.3212890625, -0.93603515625, -1.28125, -1.025390625, 0.10406494140625, 1.6884765625, -0.67431640625, -0.61376953125, -0.646484375, 0.380615234375, -1.1259765625, -1.2958984375, 0.73046875, -0.6513671875, -0.084716796875, 0.64892578125, -0.22705078125, -0.56884765625, 0.8544921875, -0.69580078125, 0.771484375, 0.65087890625, 0.93701171875, -0.438720703125, -0.75830078125, 0.1668701171875, -0.1805419921875, -1.02734375, 1.0224609375, -0.90234375, 0.246337890625, -0.004608154296875, 0.6728515625, -0.62451171875, -0.34375, -1.62890625, -0.018524169921875, -0.06427001953125, -0.326416015625, 1.4755859375, 0.701171875, 1.0390625, -0.013336181640625, 1.2900390625, -1.3994140625, 1.4755859375, 0.333251953125, -0.41455078125, 0.35107421875, 0.177490234375, -0.603515625, 0.71875, 0.432861328125, 2.19140625, 0.42529296875, -0.83837890625, 0.158447265625, -1.1806640625, -0.96240234375, -1.4365234375, 0.051422119140625, -0.91650390625, -0.209716796875, 0.595703125, 1.8984375, -0.1710205078125, 0.77978515625, 0.783203125, -0.83984375, -0.0406494140625, 0.385986328125, -0.8212890625, -0.83642578125, -0.031768798828125, 1.142578125, -0.75830078125, 0.447265625, -0.6025390625, 1.814453125, -1.0703125, 0.177490234375, -0.98046875, 1.2578125, 0.3330078125, -0.5341796875, -0.041046142578125, 1.212890625, -0.0261383056640625, 0.51318359375, -0.2646484375, -0.262451171875, -0.56884765625, -1.6904296875, -1.4970703125, 0.59326171875, 0.130859375, -0.0672607421875, -0.28466796875, 0.15869140625, 0.8642578125, 0.8330078125, 0.5078125, 0.3408203125, -0.53369140625, -0.7158203125, -0.38330078125, -0.84912109375, -0.377197265625, 0.6435546875, -0.67626953125, 0.1387939453125, 0.007781982421875, 0.61865234375, 0.7275390625, -0.34619140625, 0.54052734375, 0.86572265625, 0.63818359375, 0.20458984375, 0.89013671875, 1.5654296875, -0.339599609375, 0.1346435546875, -0.10400390625, -0.59033203125, -0.7783203125, -0.425048828125, -1.740234375, 0.7216796875, -0.1905517578125, -2.12890625, 0.26171875, 1.04296875, 0.3798828125, 0.274169921875, 0.9833984375, -0.16796875, 1.2216796875, -0.2548828125, 1.59765625, 1.046875, 0.89892578125, -0.39990234375, -0.72265625, -1.345703125, -0.35693359375, -0.254638671875, -1.0615234375, 2.458984375, 0.9296875, -0.259033203125, 0.0021114349365234375, -1.0908203125, 0.29638671875, 0.2486572265625, -0.67138671875, -0.3154296875, 1.0888671875, 0.29638671875, -0.212890625, -0.425048828125, -0.2802734375, -0.235595703125, -0.4462890625, 0.450927734375, -0.330322265625, 1.8203125, 0.62060546875, 0.256591796875, -0.162109375, -1.6494140625, 1.873046875, -0.76904296875, 0.66162109375, 1.4609375, 0.455322265625, 0.72900390625, -1.6650390625, -0.453857421875, 1.451171875, 0.994140625, 1.0400390625, 0.716796875, -0.96240234375, 0.383544921875, 1.330078125, -1.2607421875, 2.6640625, -1.3427734375, -0.6728515625, -0.41552734375, 0.6376953125, -1.09375, -0.15576171875, -0.1983642578125, -1.6337890625, 0.056549072265625, -0.64892578125, -0.64404296875, 0.056365966796875, 0.7431640625, -0.7890625, -0.82763671875, -0.94384765625, 0.939453125, -0.85595703125, 0.4443359375, -0.046905517578125, 1.2919921875, -0.8203125, 0.31396484375, 0.341796875, -0.0816650390625, -0.12127685546875, 0.7568359375, -1.6875, 2.1953125, 0.394775390625, -0.8671875, 0.7314453125, -1.1630859375, -0.378662109375, -0.059478759765625, 0.0262603759765625, -1.404296875, -0.66552734375, 0.448974609375, -0.0694580078125, -1.3369140625, 0.69580078125, -1.046875, -0.145263671875, -1.908203125, 1.1826171875, -0.407470703125, 1.03125, 0.4814453125, -0.042755126953125, -0.7470703125, 0.69384765625, 0.1165771484375, 1.1064453125, -0.53515625, -0.09063720703125, 0.951171875, 1.4130859375, -0.5380859375, -0.94873046875, 0.60009765625, -0.155517578125, -2.11328125, -1.4462890625, 0.0926513671875, 0.174072265625, 1.6728515625, 1.12109375, -1.6455078125, 0.0179290771484375, 1.19921875, -0.3193359375, -0.0208892822265625, 1.00390625, -0.00617218017578125, -0.80029296875, -0.7822265625, 0.84912109375, -1.0830078125, 0.142333984375, -0.318603515625, -0.1517333984375, 0.91943359375, -0.439453125, 1.0927734375, 0.308837890625, -0.76416015625, 1.306640625, -0.26953125, -1.8095703125, 1.708984375, 1.4169921875, -0.052947998046875, 0.677734375, 0.454345703125, -0.494384765625, 1.1083984375, -0.421875, 0.228515625, -0.5380859375, 0.6611328125, -0.36962890625, 0.54248046875, 0.37548828125, -0.103271484375, 2.748046875, -1.7978515625, 1.111328125, -0.78125, 2.001953125, -0.5166015625, -0.93603515625, -2.052734375, -1.189453125, 0.7314453125, -0.2314453125, 0.52880859375, 0.38427734375, 0.09100341796875, -1.5048828125, -0.55224609375, -0.50048828125, 1.453125, 0.55419921875, -0.125732421875, 0.40869140625, -2.1328125, -1.6376953125, 1.9013671875, -1.94140625, -1.052734375, -1.0234375, 0.1832275390625, -2.29296875, -0.6279296875, -0.447265625, 0.2210693359375, 1.4150390625, -2.748046875, -0.84375, 0.1260986328125, -0.70458984375, 0.822265625, -0.61376953125, -1.888671875, -0.457275390625, 0.06341552734375, -0.42724609375, 1.25, 0.6787109375, 0.061065673828125, -1.408203125, 0.019256591796875, -2.90625, -1.1181640625, -0.82421875, 2.603515625, -0.135498046875, 0.140869140625, 0.8837890625, -1.34375, 0.0037937164306640625, 1.328125, -1.2392578125, 1.1376953125, 0.1728515625, -1.55859375, 0.06744384765625, -0.98828125, 0.58740234375, -1.1201171875, 0.82958984375, -0.58935546875, -0.82080078125, 1.2666015625, -0.204345703125, 0.47607421875, -0.76318359375, 0.78759765625, 0.52197265625, 0.446044921875, 0.401611328125, -0.388427734375, 0.0300750732421875, -0.29345703125, -0.036102294921875, 1.16015625, -0.28662109375, -3.041015625, -0.328857421875, -1.1337890625, -0.58935546875, 0.48046875, -1.078125, -1.7822265625, 0.478271484375, 1.603515625, 0.6201171875, 0.282958984375, 0.8720703125, 0.014862060546875, 1.513671875, 1.40625, 0.5576171875, -0.462890625, 0.66357421875, -0.54931640625, 0.2176513671875, -0.05023193359375, -0.609375, -1.306640625, -1.61328125, -1.3017578125, -1.2724609375, 0.1962890625, 1.5908203125, 1.1845703125, 0.293212890625, 2.068359375, 0.287109375, 0.338134765625, 1.0966796875, 0.181396484375, -0.349609375, 0.472412109375, 1.181640625, 0.5673828125, -0.328857421875, -0.09039306640625, 1.4755859375, 0.01291656494140625, 2.4921875, -1.93359375, 0.48046875, 0.08837890625, -0.311279296875, 0.67626953125, -0.68310546875, 0.146240234375, -0.77099609375, -0.6259765625, -1.55859375, 1.396484375, -1.1416015625, 0.7470703125, 0.028533935546875, -0.421142578125, 0.81640625, 1.103515625, -1.2353515625, -0.67236328125, 0.1297607421875, 1.1064453125, -0.040863037109375, 1.2744140625, -1.421875, -0.467041015625, -0.62841796875, 0.32470703125, -0.6748046875, 0.75439453125, 0.283935546875, 0.90673828125, 1.6396484375, -0.1680908203125, -1.0390625, -1.5029296875, 1.578125, -0.1927490234375, 1.484375, -0.689453125, 0.546875, 1.0986328125, -0.177001953125, 1.583984375, 1.7412109375, 0.2222900390625, -0.90380859375, -1.216796875, 1.3095703125, -1.205078125, 1.3759765625, -0.2271728515625, 0.07666015625, -2.107421875, -0.861328125, 0.3720703125, 1.037109375, -1.5537109375, 1.1181640625, 0.9697265625, 1.7783203125, -2.255859375, 1.0146484375, -1.7958984375, -1.22265625, -1.73046875, -0.41064453125, 0.427490234375, 1.64453125, 1.7958984375, 0.083251953125, 0.060791015625, -0.00276947021484375, 1.16796875, -0.54931640625, 1.21875, 0.751953125, 0.80419921875, 0.39306640625, -0.418701171875, 0.79736328125, -0.1380615234375, 1.73828125, 0.66552734375, 0.15185546875, 1.337890625, -0.41162109375, -0.29248046875, 0.27734375, -0.69677734375, -0.5234375, 0.6259765625, -0.5244140625, 0.97607421875, 1.35546875, 0.59033203125, -0.031829833984375, -0.58203125, 0.513671875, -0.27001953125, 0.040130615234375, 0.7998046875, 0.294921875, -0.69921875, -0.6494140625, -0.486572265625, 0.326904296875, 0.35546875, 0.104736328125, -0.74267578125, 1.13671875, -1.05078125, -0.485595703125, 0.2110595703125, -0.101806640625, 0.403076171875, 1.140625, -0.697265625, -0.56640625, 0.4453125, -0.42236328125, 0.9169921875, -1.4638671875, 0.919921875, 0.293701171875, -1.09765625, 1.2001953125, -0.81005859375, 0.357666015625, 0.65673828125, -1.1875, 0.01494598388671875, 0.18115234375, -2.025390625, -0.78076171875, 1.900390625, 0.546875, -1.7763671875, -0.84375, -0.66357421875, -1.1689453125, -0.95263671875, 0.50341796875, 1.1474609375, -1.107421875, -0.354248046875, -0.509765625, -0.32666015625, -0.0321044921875, -0.181884765625, -2.447265625, -0.65283203125, 0.99169921875, 0.1424560546875, 0.77294921875, 0.6201171875, -0.7529296875, -0.72802734375, 0.82373046875, 0.68701171875, 0.434326171875, 0.544921875, 1.1435546875, -1.3017578125, -0.11187744140625, -0.2271728515625, -0.6083984375, 0.66552734375, 0.393798828125, 0.58544921875, -0.556640625, 0.193603515625, 0.95556640625, -0.5224609375, 0.1951904296875, -0.441650390625, -1.3525390625, 0.99365234375, -0.5439453125, -0.70751953125, 1.783203125, -2.0546875, 0.70947265625, -1.9501953125, 0.1763916015625, 0.25341796875, -0.265869140625, 0.1514892578125, 1.705078125, 0.75341796875, -0.953125, 1.66015625, 0.0223236083984375, -0.08673095703125, 0.08599853515625, 1.2138671875, -0.81982421875, 0.4365234375, -0.0261383056640625, 0.56982421875, -0.68115234375, -0.0673828125, -1.9072265625, -0.6875, 0.85009765625, -1.736328125, 0.236572265625, 0.16259765625, -0.2423095703125, 0.07611083984375, 0.371337890625, -1.0263671875, 1.9296875, 0.9453125, 0.389404296875, -1.279296875, 0.1807861328125, -0.0693359375, -1.345703125, -0.5107421875, 0.045654296875, 1.021484375, 0.0670166015625, -2.263671875, 0.69384765625, -1.1318359375, -0.369384765625, 0.3984375, -0.2374267578125, -0.09527587890625, 2.15234375, -0.139404296875, -0.9970703125, -1.724609375, 1.080078125, -0.63916015625, -1.03515625, -0.5068359375, 0.408203125, -1.1103515625, 0.91455078125, -1.541015625, 0.022979736328125, 0.366455078125, -0.343994140625, 0.70849609375, -1.013671875, 0.64794921875, 0.06439208984375, 0.0726318359375, -0.498779296875, 0.61474609375, 0.466552734375, 0.98828125, -1.0263671875, -0.99072265625, 0.0031490325927734375, -1.5234375, 0.94970703125, 1.9130859375, -0.626953125, -0.2119140625, -0.9150390625, 3.224609375, -0.1015625, 0.9150390625, -1.7099609375, 0.0357666015625, 0.2027587890625, -0.288330078125, 0.1788330078125, -2.06640625, -2.59375, -0.62158203125, -0.409912109375, 1.5478515625, -0.658203125, 0.3837890625, 2.794921875, -0.69189453125, 1.75, 1.6416015625, 1.892578125, -0.241943359375, 0.60009765625, 1.5830078125, -0.89794921875, 0.8203125, -0.92578125, -1.0009765625, 1.4404296875, -1.1982421875, -0.45458984375, -0.7392578125, 0.01580810546875, -1.958984375, 0.52294921875, 1.39453125, -1.1181640625, -0.51416015625, -1.3828125, 0.263671875, 0.267333984375, -1.2724609375, 0.6455078125, -0.91650390625, 2.076171875, 0.283935546875, 1.216796875, 0.1180419921875, 1.32421875, -0.2120361328125, 0.256591796875, -0.685546875, -0.87158203125, 0.1312255859375, -2.166015625, -1.61328125, 0.6640625, 0.8095703125, -0.9765625, -0.83349609375, 1.0908203125, -0.94482421875, 0.431396484375, 0.67431640625, -0.3388671875, 2.078125, -1.5908203125, -0.1744384765625, 1.875, 1.23046875, -1.994140625, -0.38671875, -0.11859130859375, -1.5078125, -0.1849365234375, 0.1890869140625, 0.215576171875, -1.3330078125, -0.87353515625, 0.455810546875, 0.031890869140625, 0.71435546875, -0.06500244140625, -0.53955078125, 0.2447509765625, 0.48681640625, 0.5087890625, -0.578125, -1.4384765625, 0.8037109375, -1.080078125, -2.205078125, 1.2080078125, 0.92333984375, 1.447265625, 0.60400390625, 0.88671875, 0.7607421875, 0.478271484375, 0.01531982421875, 0.174072265625, 1.763671875, 1.8896484375, -1.6630859375, 0.202880859375, 1.3134765625, -0.80615234375, -0.39697265625, 0.2744140625, -0.8173828125, -1.861328125, -2.34375, 0.04351806640625, 0.6240234375, 0.390625, -1.1181640625, 0.306640625, -0.1865234375, 2.064453125, -1.7177734375, 0.92724609375, -0.2237548828125, 2.974609375, -2.26171875, -0.2354736328125, 1.7978515625, -0.5546875, -0.1766357421875, 0.394775390625, 0.1990966796875, 0.1650390625, -1.6611328125, 0.50537109375, 0.13134765625, 0.0665283203125, -1.3291015625, 0.12890625, -0.4111328125, -0.85302734375, -1.3818359375, 1.69140625, 0.51904296875, -0.376708984375, 1.4375, -1.12890625, 0.8662109375, -1.3701171875, -0.501953125, -0.7119140625, -1.056640625, -1.419921875, -1.4951171875, -0.64697265625, -1.517578125, 1.6279296875, 0.23095703125, 0.04425048828125, 0.2802734375, 0.270751953125, 0.323974609375, -1.9521484375, 0.38720703125, 0.2010498046875, -1.0205078125, 0.67822265625, -0.64599609375, 1.2744140625, 1.111328125, -0.251220703125, 0.2449951171875, -0.54345703125, 2.12109375, 0.2469482421875, 0.30810546875, -0.5029296875, 0.9189453125, 1.2568359375, -0.984375, 0.1578369140625, -1.568359375, -0.80419921875, -0.9716796875, -0.447509765625, 0.289794921875, -0.12261962890625, -0.51416015625, -0.84765625, -0.28515625, -0.70556640625, -0.1810302734375, 1.8154296875, -0.55322265625, -1.208984375, 0.74560546875, -0.2393798828125, -0.1474609375, -1.4365234375, -0.0103912353515625, 1.2001953125, -2.837890625, -0.1412353515625, 0.96826171875, -0.3740234375, 0.81689453125, 0.74853515625, 0.43310546875, -0.6650390625, -1.595703125, 1.5498046875, -1.0751953125, 1.36328125, -0.459716796875, -1.9462890625, -0.0845947265625, 1.798828125, 0.274169921875, 0.10626220703125, 0.470458984375, 0.69677734375, -0.0867919921875, -1.78125, -1.8076171875, 1.322265625, -0.380126953125, -1.10546875, 0.2073974609375, 1.4873046875, 1.6298828125, -1.4462890625, -1.0966796875, -0.57275390625, -0.875, -1.5224609375, -0.54541015625, 1.673828125, 0.61669921875, -0.9697265625, -0.4033203125, -0.50927734375, -0.7275390625, -0.3427734375, 1.71875, -0.454833984375, -1.92578125, 1.17578125, -0.1824951171875, 1.46484375, 1.759765625, 1.296875, -0.340087890625, 1.4990234375, -0.86865234375, -0.91162109375, 0.2210693359375, 0.6591796875, -0.346435546875, 0.46923828125, -0.037078857421875, 0.46044921875, 0.34228515625, -0.2183837890625, 1.3740234375, 0.68017578125, 2.056640625, 0.40478515625, -0.53271484375, 1.9091796875, 0.763671875, 0.013580322265625, -0.429931640625, -0.0228271484375, 0.6318359375, 0.60546875, 0.0204315185546875, 0.90869140625, -0.60400390625, -0.61181640625, 0.0728759765625, 1.9462890625, 0.96044921875, 1.3408203125, -0.08868408203125, 0.78662109375, 0.176513671875, 0.755859375, -1.25, -1.0166015625, -0.5048828125, 0.043609619140625, -0.255859375, 0.0718994140625, 0.03826904296875, -0.23828125, -0.363525390625, -0.9970703125, -0.5087890625, 1.40234375, 1.44921875, -0.489013671875, 0.1751708984375, 0.7724609375, -0.2763671875, 0.326171875, -0.6806640625, -0.382080078125, 0.48193359375, 1.7158203125, -1.009765625, 0.634765625, -1.2041015625, -0.1314697265625, 0.71435546875, -0.382568359375, -0.039794921875, 2.009765625, -0.262451171875, 0.62158203125, -0.5947265625, 0.1888427734375, 0.517578125, 0.54150390625, -1.7119140625, 0.31396484375, -1.62890625, -0.1904296875, 0.177490234375, -0.2091064453125, 0.168701171875, 1.150390625, 0.677734375, -0.27685546875, 1.7470703125, -0.362548828125, 1.3310546875, -0.72265625, 0.296142578125, 0.06683349609375, 0.806640625, 2.017578125, 0.787109375, 0.2724609375, -0.841796875, 0.30615234375, 0.322265625, -0.7841796875, -0.29833984375, 0.0728759765625, 0.10491943359375, 1.81640625, -1.6240234375, 1.2744140625, -1.3349609375, -0.308349609375, 0.042724609375, 1.78515625, 0.360107421875, 1.9736328125, -0.91650390625, -1.1337890625, -0.78076171875, 1.0869140625, -1.119140625, 1.78125, -1.365234375, -0.59326171875, -0.47607421875, -0.8359375, 0.98876953125, 0.50439453125, 0.94189453125, 0.257568359375, -0.1678466796875, 1.7109375, -1.3974609375, 0.185791015625, -2.380859375, -0.32275390625, 0.9365234375, -0.56591796875, 0.319091796875, -0.259521484375, -0.1126708984375, -0.86962890625, 0.91162109375, 0.054229736328125, -0.301025390625, -1.556640625, -1.6845703125, -0.40185546875, 1.1494140625, 0.626953125, -0.67724609375, -0.7998046875, -0.20556640625, -1.681640625, 0.97265625, 0.0875244140625, 1.8125, -0.67333984375, 0.93798828125, -0.513671875, 0.00948333740234375, -0.658203125, -0.342041015625, -1.3818359375, 1.7373046875, -0.337646484375, 0.71826171875, -1.2470703125, -0.52783203125, -0.324951171875, 0.52978515625, 0.329833984375, -0.99267578125, -0.55078125, 0.9580078125, -0.264892578125, -0.6259765625, -0.92529296875, -0.7578125, -0.356689453125, 2.3984375, -0.07818603515625, 0.31298828125, 1.4560546875, -0.35009765625, 0.99560546875, -0.487060546875, -0.365478515625, -0.2435302734375, -0.107666015625, 1.591796875, -0.28564453125, 1.4599609375, -0.65234375, 0.70849609375, 1.9970703125, -0.28759765625, 1.6943359375, 0.7802734375, -0.65185546875, 0.0034503936767578125, 1.9609375, 0.245849609375, -0.0718994140625, -1.771484375, 1.236328125, 1.5185546875, -0.46240234375, 0.2418212890625, -0.01251983642578125, -1.642578125, 0.317138671875, 0.468017578125, -0.94091796875, -0.0684814453125, 0.81689453125, 0.364990234375, -1.4404296875, 0.280517578125, 1.1591796875, -1.9052734375, -0.450927734375, -0.7783203125, -0.54931640625, -0.345703125, 0.6494140625, -0.44189453125, 0.853515625, 1.7841796875, 1.357421875, -0.418701171875, -0.8515625, -0.03271484375, -0.31591796875, -1.28515625, 0.505859375, -0.8017578125, 0.18798828125, -0.017059326171875, -0.7421875, 0.1583251953125, 1.87890625, -1.34375, -0.890625, -0.3759765625, 0.0716552734375, 0.8876953125, 1.1025390625, 0.7275390625, -0.37255859375, 1.1201171875, -0.296142578125, 0.85498046875, 0.229736328125, 1.185546875, -0.55322265625, 1.6630859375, -0.2232666015625, 0.7490234375, -1.69140625, 0.52001953125, 1.681640625, -0.7685546875, -1.1591796875, -1.748046875, 0.3046875, 0.434814453125, -0.259765625, 0.7021484375, 2.162109375, 0.50341796875, -0.468505859375, 1.3984375, 1.4365234375, 0.41357421875, -0.052886962890625, -1.3916015625, 1.98046875, -0.2476806640625, -0.68505859375, 0.8369140625, -0.486328125, -0.7265625, 0.23486328125, 0.93603515625, -2.95703125, 0.22314453125, 0.454345703125, -3.24609375, -1.3701171875, 0.595703125, 0.052581787109375, -0.13525390625, 2.0, 0.44970703125, -1.7412109375, -0.470947265625, -0.131591796875, -0.08709716796875, 0.26025390625, 0.06878662109375, -1.8232421875, 1.5458984375, 1.37890625, 0.791015625, -0.057403564453125, -1.1318359375, -1.9736328125, -0.51953125, 0.2578125, 0.470458984375, -0.9345703125, -1.0625, -0.48583984375, 0.130126953125, 0.6923828125, -0.00815582275390625, 0.572265625, 0.21142578125, 0.0030155181884765625, -0.87890625, 1.4921875, 1.0615234375, 0.398193359375, -0.08380126953125, -1.513671875, 1.2421875, 0.57568359375, 0.2271728515625, 1.083984375, 0.8388671875, 0.2059326171875, 0.1588134765625, -0.52685546875, -2.16015625, -0.76025390625, 0.0673828125, -1.216796875, 1.8232421875, -1.2744140625, -0.09600830078125, 0.3330078125, 0.68603515625, -0.188232421875, 0.08013916015625, 0.1893310546875, 0.7724609375, 0.85791015625, 0.48828125, -0.418212890625, -0.61376953125, 0.75830078125, -0.8896484375, -0.3671875, 0.31103515625, 1.7275390625, -0.64599609375, -0.759765625, 0.09063720703125, 0.41455078125, 0.360107421875, 0.493896484375, 0.25341796875, 0.712890625, -0.712890625, -0.58935546875, -1.1298828125, 0.46728515625, -0.6865234375, 1.673828125, -0.281982421875, 0.6953125, -0.3671875, -1.0126953125, -2.1171875, -0.181640625, -0.049224853515625, 0.11273193359375, 1.0888671875, -0.1260986328125, 0.2431640625, -1.1123046875, -0.034393310546875, -0.04046630859375, 0.740234375, 0.05108642578125, 0.83203125, -0.09515380859375, 0.13818359375, -0.2281494140625, -1.0498046875, -0.0189056396484375, -1.353515625, -1.9453125, 1.001953125, 0.67041015625, 2.396484375, -0.434326171875, 1.8701171875, -0.40380859375, -0.17724609375, 0.37060546875, 0.84765625, -0.2432861328125, -0.9306640625, -1.1689453125, 0.137939453125, 0.148681640625, -1.0078125, -0.7451171875, -1.8056640625, -1.8525390625, -1.15625, -1.1689453125, -0.6513671875, 1.3896484375, -2.330078125, 0.7802734375, 0.3115234375, 1.3564453125, -1.0478515625, 1.1494140625, 0.90869140625, -0.63623046875, 0.54248046875, -0.7265625, 1.095703125, 1.125, -0.157958984375, -0.74365234375, -1.244140625, -1.16015625, -0.1715087890625, 0.365234375, -0.873046875, -0.492431640625, -0.74072265625, 0.58935546875, -2.119140625, -0.66455078125, -0.96240234375, 1.3759765625, 0.88623046875, -0.49169921875, 1.1328125, -0.33984375, 0.04278564453125, -3.275390625, -0.01192474365234375, 0.7138671875, -0.72265625, 0.48974609375, -1.5126953125, 0.453857421875, 0.160400390625, -0.1072998046875, 2.080078125, 0.1300048828125, 0.62451171875, 2.033203125, 0.88525390625, 1.1083984375, 0.25439453125, -1.04296875, -1.521484375, 0.1878662109375, 0.454833984375, 1.6123046875, -0.7265625, 0.57275390625, -0.0207672119140625, -0.755859375, -1.720703125, -1.0751953125, -0.6318359375, -1.4794921875, 1.359375, 0.01261138916015625, 1.0966796875, -0.68701171875, -0.044403076171875, -0.47216796875, 1.7265625, 0.71435546875, 1.0712890625, 1.11328125, -1.2021484375, -1.18359375, -0.5498046875, 0.423828125, -2.23046875, -1.2578125, -1.3515625, -1.369140625, 0.95556640625, 1.1220703125, -0.2406005859375, -0.5107421875, -0.451171875, 2.009765625, -0.322265625, 0.51708984375, 0.56787109375, 1.43359375, -1.0478515625, 2.09765625, 0.202880859375, 0.1715087890625, 0.1285400390625, 0.480224609375, -0.80615234375, 0.0628662109375, -0.267822265625, -0.98095703125, 0.71337890625, 1.2392578125, 2.587890625, 0.82763671875, 0.129150390625, -1.1962890625, 0.006801605224609375, 1.462890625, 0.6494140625, 1.005859375, 0.09564208984375, -1.59375, 1.2509765625, -0.8037109375, 0.65234375, 0.07855224609375, -0.271728515625, 1.9560546875, 1.5595703125, -1.623046875, -1.947265625, -0.31201171875, -0.81103515625, -0.99658203125, -0.45703125, -1.0986328125, -1.662109375, -1.150390625, 0.0126953125, 0.468017578125, -0.89990234375, -0.447998046875, 0.81396484375, 1.1552734375, -0.3232421875, -2.013671875, 0.01250457763671875, -1.3564453125, -1.2197265625, 0.3955078125, -0.0574951171875, -0.5888671875, -0.266357421875, -0.44921875, -0.5048828125, -0.88525390625, -0.0157470703125, -1.0263671875, 1.0322265625, -0.203125, 0.305419921875, 0.83251953125, 1.310546875, 0.1851806640625, 0.06781005859375, -1.0712890625, 1.353515625, 0.172607421875, -0.7880859375, 0.1845703125, 0.5751953125, -0.04705810546875, -0.328369140625, -0.7919921875, 1.3076171875, -0.5537109375, 0.85302734375, -1.2919921875, 0.429931640625, 1.62890625, -1.40234375, -0.053314208984375, 0.56298828125, 0.07269287109375, 2.888671875, -0.491455078125, 0.51904296875, 1.109375, 0.53564453125, 0.408447265625, -2.107421875, 0.64111328125, 2.01171875, 0.09735107421875, 0.810546875, -1.5732421875, 1.220703125, -2.138671875, -1.65625, 0.489013671875, -0.6796875, 1.2197265625, 0.92578125, 0.0657958984375, -0.1954345703125, 0.568359375, 0.77294921875, -2.921875, 0.059814453125, 0.7197265625, -0.0211029052734375, 1.779296875, -0.7939453125, 0.56640625, 0.440673828125, 0.47509765625, 0.46142578125, 1.30078125, -2.462890625, -0.016082763671875, 0.059295654296875, 0.5107421875, 0.364013671875, -1.302734375, -1.73828125, -0.97021484375, 0.2822265625, 0.6982421875, 0.218505859375, 1.3583984375, -0.84619140625, -1.5927734375, -0.333251953125, 0.77197265625, 0.1639404296875, -0.1036376953125, 0.7060546875, 1.0732421875, -0.97705078125, -1.36328125, 1.5498046875, 1.0498046875, 1.900390625, 0.9892578125, 0.363525390625, -2.046875, 0.939453125, -2.533203125, 1.416015625, 1.494140625, -0.58642578125, -1.08203125, 0.7021484375, -1.220703125, -0.262451171875, 2.6640625, -0.04547119140625, 0.26318359375, -1.1142578125, -0.479248046875, -0.03118896484375, -0.08062744140625, -0.263671875, 1.7978515625, -0.583984375, 0.55615234375, -0.41796875, 2.015625, -0.5634765625, -1.7763671875, 1.9208984375, -0.23095703125, -2.384765625, 0.5517578125, 0.93896484375, 0.82177734375, -0.46484375, -1.2001953125, 1.05078125, 0.41259765625, -0.57470703125, -0.07513427734375, -0.019866943359375, -0.85986328125, -0.305908203125, -0.4345703125, -0.72314453125, 2.10546875, -0.79052734375, 0.36328125, 1.40234375, 0.603515625, -1.322265625, -2.021484375, 0.61572265625, 0.65576171875, 0.546875, -1.798828125, 0.50244140625, -0.2626953125, 0.007274627685546875, 1.4150390625, 0.3583984375, -0.58642578125, 0.031890869140625, -2.1015625, -0.642578125, 0.0885009765625, 0.184326171875, -0.56494140625, 0.09588623046875, -0.265625, -0.25146484375, 1.9677734375, 0.87841796875, -0.63427734375, 1.0537109375, -0.58056640625, 0.318115234375, -0.43212890625, 0.78125, 2.005859375, -0.61279296875, 1.138671875, -1.0673828125, 0.20654296875, -0.462646484375, -0.344970703125, 0.396728515625, -0.5576171875, 0.67333984375, 0.041717529296875, -0.351806640625, 0.46826171875, 0.2418212890625, -0.00774383544921875, -0.397216796875, -1.0205078125, -0.9697265625, -1.134765625, 2.13671875, 0.779296875, 0.38720703125, -0.6650390625, 2.625, 0.58056640625, -1.0859375, 0.45703125, 1.265625, 0.254638671875, 1.0029296875, -1.9658203125, 0.5986328125, 0.213134765625, -1.130859375, 1.529296875, -1.2685546875, -0.93017578125, 2.302734375, -2.12890625, -0.101806640625, -1.0380859375, 1.1806640625, 1.884765625, 1.64453125, -0.79541015625, 0.60546875, -1.533203125, 0.80126953125, 0.58056640625, -0.8759765625, 1.056640625, 1.3837890625, 1.533203125, 0.027313232421875, 0.96533203125, 2.021484375, -0.541015625, 2.134765625, 0.935546875, -0.1944580078125, -0.82763671875, -1.8544921875, -1.248046875, 0.4951171875, -0.212890625, 1.7568359375, -0.2401123046875, -0.0335693359375, -0.890625, -0.67041015625, 1.5380859375, -0.96240234375, -0.441650390625, -0.56640625, -0.07623291015625, 1.5244140625, -0.86669921875, -0.449462890625, -1.181640625, -0.138427734375, 0.2437744140625, -0.5810546875, -0.2098388671875, 0.73974609375, 0.354736328125, 0.32470703125, 0.337890625, 0.01702880859375, 0.2132568359375, 0.8818359375, 0.75244140625, -0.002185821533203125, 0.76318359375, -0.572265625, -2.994140625, 0.6572265625, -0.5908203125, -1.0341796875, -0.94287109375, 0.31591796875, -0.455322265625, 0.43701171875, -0.386962890625, -0.338623046875, -0.619140625, -0.81396484375, 0.0379638671875, 1.5, -0.381591796875, -0.386962890625, 0.97900390625, -0.5166015625, 0.48681640625, 1.2470703125, -2.224609375, -0.037078857421875, -0.62939453125, 1.1171875, -0.478515625, -0.001621246337890625, 1.392578125, 0.260009765625, 1.5341796875, 0.2255859375, 0.03338623046875, 0.1497802734375, -1.271484375, -0.8662109375, -0.5869140625, -1.3779296875, 1.55859375, -0.63232421875, 1.837890625, 0.404541015625, -1.724609375, -0.59326171875, -1.4208984375, 0.56201171875, 1.072265625, 0.448486328125, 1.3369140625, -1.7314453125, 0.7001953125, -1.4736328125, 0.23681640625, 0.231689453125, 0.78173828125, -1.0947265625, 1.1982421875, 0.40869140625, 0.66748046875, 1.0869140625, -2.080078125, 1.4248046875, -2.009765625, 1.5458984375, -0.493408203125, -1.4189453125, -1.447265625, 0.023681640625, -0.55322265625, -1.1640625, -1.2626953125, 0.429443359375, 1.09375, -0.056182861328125, -0.52880859375, 0.471923828125, 0.0625, -0.288330078125, -0.43994140625, 0.447509765625, -1.27734375, 0.2001953125, -0.408447265625, -0.78125, 0.5078125, 0.369140625, -0.52783203125, 2.333984375, -1.1884765625, -1.4150390625, -1.888671875, -0.27587890625, -0.787109375, 0.06591796875, 0.059356689453125, -0.13720703125, -0.294189453125, -0.274169921875, -0.056671142578125, 0.1412353515625, -1.1318359375, -1.4375, 0.10565185546875, 0.9638671875, 0.83984375, -0.921875, -0.499267578125, 0.08306884765625, 0.483642578125, 0.88916015625, -0.45556640625, -0.77099609375, 0.121337890625, 0.53857421875, -0.0231781005859375, -0.521484375, 0.34326171875, 1.4501953125, -0.5400390625, -0.2008056640625, -0.1029052734375, 2.0390625, 0.39892578125, -1.0380859375, -1.2509765625, 1.1435546875, -0.53369140625, 0.358642578125, 0.1156005859375, -1.0380859375, 0.050994873046875, 0.398681640625, -0.9228515625, 0.0906982421875, -0.1287841796875, -1.8359375, -1.900390625, -0.6357421875, 0.484130859375, 0.994140625, 0.104736328125, 0.56689453125, 0.78662109375, 0.67578125, -0.70166015625, 0.509765625, -0.296630859375, 1.208984375, -0.90087890625, -2.228515625, 0.384765625, 1.697265625, -0.0933837890625, -1.341796875, 0.5634765625, 1.0791015625, 0.65771484375, -0.60400390625, -1.9658203125, -0.41162109375, 0.242431640625, -1.7802734375, 0.364990234375, -0.06903076171875, 0.1337890625, -1.654296875, -0.681640625, -0.74462890625, -0.37060546875, -1.70703125, 0.440185546875, 0.1339111328125, 0.460693359375, -1.259765625, -0.96533203125, 1.5634765625, 0.0977783203125, 1.3359375, -0.34765625, -0.004573822021484375, 0.6142578125, -0.1114501953125, -1.5888671875, 0.14013671875, -1.1220703125, 0.458251953125, -1.7548828125, 0.044158935546875, -0.403076171875, -1.2099609375, -0.265869140625, 0.495849609375, -1.6806640625, -1.646484375, -1.3193359375, -0.82421875, -1.01953125, 0.435546875, 0.67138671875, -0.2822265625, -0.607421875, 0.6083984375, 1.05078125, -0.02032470703125, -1.560546875, 0.6357421875, -0.5810546875, 0.276123046875, -1.68359375, 0.0014209747314453125, 0.48486328125, 2.6796875, 0.192138671875, -1.546875, -0.29638671875, -2.29296875, -0.26953125, 0.0190582275390625, 1.23046875, 1.0810546875, 0.410400390625, 2.306640625, -1.80859375, 0.499267578125, 1.271484375, -0.412109375, 0.145751953125, 1.208984375, 0.1866455078125, 1.974609375, 1.7470703125, 0.1239013671875, -0.004825592041015625, -0.82177734375, -1.3427734375, -0.37060546875, -0.96533203125, 0.6865234375, -0.147216796875, -0.623046875, 0.2034912109375, -0.30908203125, 0.4814453125, -0.56201171875, -0.40869140625, -1.84375, 1.96484375, -0.89794921875, -0.77783203125, -0.06292724609375, 0.91845703125, -1.06640625, 0.724609375, -0.416748046875, -0.17919921875, 0.705078125, 1.1484375, -1.4990234375, -0.062042236328125, -0.21337890625, -0.08648681640625, -0.9443359375, 1.2705078125, 0.80029296875, -1.287109375, 0.9091796875, 1.4052734375, 0.5556640625, 2.197265625, -0.11376953125, -1.11328125, 0.2314453125, 0.300048828125, 0.5712890625, 0.75048828125, 0.85693359375, -0.8291015625, 0.450927734375, -1.244140625, 2.158203125, -0.60498046875, 0.658203125, -0.59423828125, 1.046875, -0.5244140625, -1.4697265625, -2.21875, 1.484375, -0.1904296875, -0.334716796875, -0.410888671875, -0.10516357421875, -0.311279296875, -0.177734375, -3.380859375, 0.0031280517578125, 0.126708984375, -0.24169921875, 0.423095703125, 1.0947265625, 0.9443359375, 1.4482421875, -0.751953125, 2.783203125, 0.2310791015625, -0.169189453125, -0.7373046875, -0.036163330078125, 0.4736328125, 0.123046875, 0.67138671875, -0.4150390625, -1.3671875, -0.1893310546875, -0.25439453125, 0.0640869140625, 0.66162109375, 1.15625, -0.90380859375, 0.9521484375, -0.4365234375, 2.01953125, 0.1171875, -0.3203125, -1.935546875, -0.61328125, -1.0859375, 0.95947265625, 1.701171875, -0.071044921875, -0.58984375, -0.720703125, -0.7666015625, -0.6982421875, -0.0115509033203125, -1.1044921875, 0.5546875, 1.1669921875, -0.446533203125, 0.2255859375, -0.78515625, -0.33447265625, 0.0653076171875, 0.58544921875, 1.771484375, -2.123046875, 1.4365234375, -1.1875, 0.16357421875, -0.5693359375, -0.6435546875, 0.5458984375, -1.876953125, -0.4697265625, 0.99658203125, -2.01953125, -0.79345703125, -0.7119140625, 0.344482421875, 1.3798828125, 0.681640625, 1.1435546875, -1.12109375, 0.306884765625, 1.6494140625, 0.250244140625, -0.338623046875, 0.469482421875, -2.712890625, -1.8017578125, 0.35302734375, 1.3466796875, -0.1610107421875, 0.171875, 1.60546875, -0.2471923828125, 0.052490234375, 1.3515625, 0.6044921875, 1.2392578125, -0.475830078125, -0.3330078125, 1.0, 0.47265625, -1.1923828125, -0.9482421875, -2.228515625, 0.44482421875, 0.71728515625, -0.58251953125, -0.453125, -0.256591796875, 0.385986328125, 0.50341796875, -0.07183837890625, 1.603515625, 1.4482421875, -0.79541015625, 0.4423828125, 0.8076171875, -0.445068359375, -0.369384765625, -0.39990234375, -0.513671875, -0.8408203125, -0.005161285400390625, 1.375, 0.0648193359375, 0.52197265625, 0.253173828125, 1.685546875, -1.4013671875, 1.35546875, 0.353515625, -2.431640625, 0.24462890625, -2.251953125, 1.419921875, -1.3154296875, -0.491455078125, 0.3291015625, 0.327392578125, 0.53564453125, 0.428466796875, -0.11590576171875, 0.345703125, -1.435546875, 0.87451171875, 1.236328125, -1.8505859375, 1.359375, 2.701171875, -1.736328125, -0.2181396484375, 0.333740234375, -0.97607421875, -0.02325439453125, 2.24609375, -1.0810546875, 0.428466796875, -0.0027904510498046875, -1.12109375, 1.181640625, 0.0145416259765625, -1.4091796875, -0.28369140625, -0.1982421875, -0.2147216796875, -1.1953125, -0.1983642578125, 0.5888671875, 0.32861328125, -1.3232421875, -0.2421875, 0.0018291473388671875, 1.412109375, -0.76904296875, -0.370361328125, 0.375, -0.418212890625, -1.080078125, -0.98876953125, 0.888671875, 1.8134765625, 0.50927734375, 1.2099609375, -0.146728515625, -1.169921875, 0.027496337890625, -0.62548828125, -0.38671875, -0.42333984375, 0.6416015625, -1.0732421875, 0.841796875, 0.6103515625, -0.75244140625, 0.82373046875, -1.0439453125, 0.3671875, 0.0421142578125, -0.92724609375, 1.029296875, 0.9404296875, 0.5634765625, -0.07757568359375, 0.007709503173828125, 0.1845703125, 0.7734375, -0.47900390625, -0.576171875, -0.927734375, 1.34375, -0.2509765625, -1.2587890625, 0.98486328125, -0.49853515625, 0.1605224609375, 1.77734375, -0.75048828125, -2.71484375, -0.8984375, 1.435546875, 0.62744140625, -0.4755859375, 0.265380859375, 0.89111328125, 2.693359375, 0.1358642578125, -0.455322265625, -1.927734375, 0.90478515625, 0.48486328125, -1.927734375, -1.0263671875, 0.9951171875, 1.9609375, 0.1705322265625, 0.255615234375, 2.6015625, 0.06854248046875, 1.0732421875, 0.056640625, 0.90185546875, -1.0458984375, 1.2861328125, 2.416015625, 0.162841796875, 0.56201171875, 0.826171875, -0.7734375, -0.50048828125, 1.29296875, 2.15625, -0.3583984375, 0.36572265625, -1.3515625, -0.1373291015625, -1.4169921875, 0.3544921875, 1.1357421875, -1.7333984375, -1.2783203125, 0.59912109375, -0.73193359375, 0.310791015625, 1.044921875, 0.7470703125, 0.787109375, -0.98779296875, 1.251953125, -0.91259765625, 0.2071533203125, -1.251953125, 0.127685546875, -0.51904296875, 2.056640625, 1.619140625, -0.974609375, 0.31201171875, -1.8837890625, -0.0276336669921875, 0.43017578125, 1.2568359375, 0.71044921875, 0.333984375, -0.2578125, -0.939453125, -0.78662109375, -0.88671875, 1.6474609375, 0.7236328125, -0.1439208984375, -0.40380859375, 0.182861328125, 0.0254669189453125, -1.22265625, 1.205078125, 0.162841796875, 1.271484375, -0.8671875, 1.4423828125, 1.005859375, 0.03375244140625, -0.7353515625, -0.1944580078125, 0.61083984375, 1.0947265625, 1.02734375, -0.74609375, 0.888671875, -1.9248046875, 1.580078125, -0.326904296875, 0.32080078125, -0.321044921875, 1.41796875, 0.08184814453125, -0.8017578125, -0.048492431640625, 0.36962890625, -2.154296875, 0.42041015625, -0.3271484375, 1.2666015625, 2.638671875, -1.2080078125, 2.861328125, 1.7724609375, -1.279296875, -1.1416015625, 1.376953125, 0.8271484375, 0.018768310546875, 0.8447265625, 0.374267578125, 1.94921875, 0.6962890625, 0.06829833984375, 0.390869140625, -1.6884765625, -1.294921875, -0.65234375, -0.1658935546875, 1.177734375, 0.14404296875, -1.259765625, 0.6767578125, -0.270263671875, 0.0655517578125, 0.5791015625, 0.1866455078125, -1.943359375, 1.7822265625, -1.4443359375, -1.5419921875, -0.45703125, -0.7373046875, -0.654296875, 0.46533203125, 1.19140625, 1.15234375, 0.409423828125, 0.1314697265625, 0.697265625, 0.39453125, 0.40771484375, -0.25830078125, 1.5087890625, 1.5654296875, 1.7734375, 0.38232421875, -1.3291015625, -0.262451171875, 0.609375, 0.211669921875, -0.47119140625, 1.4482421875, 0.73486328125, -1.4716796875, 1.7734375, -0.07391357421875, 0.59716796875, 0.732421875, -0.07122802734375, -1.60546875, -0.912109375, -0.36669921875, -0.27490234375, -1.0244140625, -0.49462890625, -1.0947265625, 0.1971435546875, -1.646484375, 0.09857177734375, -0.173095703125, 1.3876953125, -1.095703125, -0.8955078125, 0.77392578125, -0.84375, -1.1787109375, -0.2156982421875, -0.48876953125, 0.07952880859375, 0.71435546875, 2.384765625, 0.27783203125, 0.2193603515625, -0.1982421875, 0.54443359375, 0.7314453125, -0.4052734375, -0.419677734375, -0.35791015625, -1.2353515625, 2.521484375, -0.57080078125, 0.397705078125, -1.8330078125, -1.0380859375, 1.8974609375, 0.29345703125, 0.5126953125, 0.280029296875, -1.0478515625, -0.79541015625, 1.01953125, 0.72119140625, -1.2822265625, -0.45849609375, -0.91259765625, 1.181640625, -0.8046875, 0.450927734375, 0.07781982421875, 1.8740234375, -0.052703857421875, 0.40234375, -0.5703125, -2.01953125, 0.4482421875, 1.001953125, -0.74658203125, -0.314453125, 0.07281494140625, -1.2744140625, -0.233642578125, -1.4609375, -1.3056640625, 0.1590576171875, 1.1767578125, 1.5517578125, -0.73779296875, -0.6318359375, 1.15625, 0.048309326171875, -0.5166015625, -1.4990234375, 1.78515625, 1.419921875, 0.91748046875, 0.039947509765625, 0.04852294921875, -0.0469970703125, 0.69921875, 0.25732421875, 0.48974609375, -0.53759765625, -1.3466796875, -0.92529296875, -1.1123046875, -0.0035839080810546875, -1.0693359375, 0.74853515625, -0.59716796875, -0.82275390625, 0.463134765625, -0.74658203125, -0.172119140625, -0.52880859375, 0.7626953125, -0.53125, -0.024566650390625, 1.7314453125, 1.5224609375, 0.92431640625, -0.9228515625, -0.26708984375, 0.513671875, 0.89794921875, -1.056640625, -0.204833984375, -0.279052734375, 0.91845703125, 0.1712646484375, 0.035552978515625, -2.498046875, 2.357421875, 0.2802734375, 0.54638671875, -0.11614990234375, 0.039031982421875, -0.853515625, -1.541015625, 0.8525390625, -0.29443359375, 0.3916015625, -0.85009765625, -0.521484375, 0.2320556640625, 0.1588134765625, -0.142578125, -1.4833984375, 0.054229736328125, 0.429443359375, -1.814453125, -0.7890625, -0.7919921875, 0.031402587890625, -0.6787109375, 0.517578125, 0.33984375, 0.13818359375, -0.67236328125, -1.4423828125, -0.55859375, 1.1259765625, -0.431884765625, 0.5634765625, 0.529296875, -0.112548828125, 0.54248046875, -2.25390625, -0.37353515625, -1.6279296875, 0.0127105712890625, -0.82177734375, 0.60400390625, -1.4775390625, -0.87109375, 1.044921875, 0.03948974609375, 0.77880859375, -1.1875, 1.54296875, 1.2734375, 1.443359375, 0.2032470703125, 1.470703125, -1.6923828125, 0.476318359375, 0.72314453125, -0.25634765625, -0.26220703125, -1.2021484375, 0.105712890625, 0.91259765625, -0.269775390625, -0.326904296875, -0.94384765625, -0.470703125, 0.9521484375, -0.861328125, 1.1728515625, 0.5810546875, -0.3251953125, -1.087890625, 0.75048828125, 0.202880859375, 1.693359375, 1.876953125, 0.17529296875, -0.98828125, -1.0107421875, 0.8515625, -0.40380859375, 0.99755859375, 0.80322265625, 0.03680419921875, -1.04296875, -1.3134765625, 0.84423828125, -1.5234375, -2.341796875, -1.849609375, 0.61865234375, 2.119140625, -0.429931640625, 0.8388671875, 1.810546875, 1.0693359375, -0.8408203125, 0.288330078125, 1.3955078125, 0.265625, -0.341552734375, -1.3359375, 0.220703125, -0.1875, 0.52001953125, -0.79052734375, -0.2216796875, -0.01141357421875, 0.9365234375, 1.33984375, 1.423828125, -1.2646484375, 0.301025390625, 0.607421875, 0.261962890625, -0.57666015625, -1.33984375, -0.07000732421875, -0.1934814453125, 0.8916015625, 0.509765625, 0.8037109375, 0.62353515625, -2.2265625, 1.119140625, -0.0205535888671875, -0.68359375, -0.08990478515625, 0.333740234375, -0.80615234375, 0.06231689453125, 0.947265625, 0.541015625, -1.25, -0.41162109375, -0.57177734375, -0.413818359375, 0.6220703125, -0.12255859375, 0.91162109375, 1.0732421875, -1.337890625, 0.64404296875, 1.50390625, -0.8212890625, 1.15234375, 2.083984375, -0.339111328125, 0.1326904296875, -0.7861328125, 0.0301971435546875, -1.158203125, -0.66064453125, -1.3046875, -1.01171875, -0.441162109375, -0.51171875, -0.74462890625, -0.80859375, -0.85546875, -1.876953125, -1.7041015625, 0.5439453125, 1.1953125, 0.37890625, -0.76318359375, 0.1309814453125, 1.1376953125, 0.5400390625, -1.42578125, 1.0888671875, -0.8076171875, 0.80078125, 0.62109375, -1.2001953125, -0.5224609375, 0.0198211669921875, -0.7294921875, 1.4130859375, -2.130859375, -0.060028076171875, -0.6259765625, 0.28271484375, 0.05364990234375, 0.39404296875, 0.166015625, -0.09381103515625, 1.0400390625, -1.123046875, 2.416015625, -0.30224609375, -0.48681640625, -1.0146484375, -0.37939453125, 1.65625, 2.955078125, 0.474365234375, 0.6220703125, -0.485595703125, -1.2265625, -0.67724609375, 0.20703125, 1.3271484375, -2.265625, 0.27587890625, 0.56689453125, 0.198974609375, -0.52099609375, -0.033782958984375, 2.681640625, 0.43408203125, -0.62548828125, 0.0262298583984375, -0.047027587890625, 0.37939453125, 0.6767578125, -0.06298828125, -0.63525390625, -0.8681640625, -0.1396484375, 0.499755859375, 1.9345703125, 0.5048828125, -0.60888671875, 1.400390625, -0.1749267578125, -1.61328125, -0.4072265625, -0.252197265625, -0.016326904296875, 1.759765625, -1.1181640625, -0.239990234375, -0.384765625, -0.276611328125, -1.2998046875, -1.421875, -1.7060546875, 0.55712890625, 0.79736328125, -0.01641845703125, -0.6865234375, 1.1015625, -0.236083984375, -1.2646484375, -0.89306640625, -1.1240234375, -0.0350341796875, 1.5673828125, 0.1607666015625, 2.013671875, -0.1624755859375, -0.276611328125, -1.9599609375, -0.35498046875, 0.65185546875, -0.5498046875, 0.233154296875, -0.248046875, 0.7353515625, -0.47216796875, -0.11761474609375, -1.1318359375, 1.2802734375, 0.794921875, -1.814453125, 0.01363372802734375, 0.5830078125, 0.70947265625, -0.81494140625, 0.31103515625, -0.79833984375, 0.93310546875, -0.489990234375, 0.135986328125, 0.03643798828125, 1.65625, -0.95458984375, 0.02081298828125, 0.79638671875, -0.410400390625, -0.33740234375, 0.399658203125, -0.84716796875, 0.375, 0.62255859375, -1.62109375, 0.36328125, 0.74609375, -0.341552734375, 0.2406005859375, -0.03472900390625, 0.274658203125, 0.59326171875, -0.6337890625, 0.53369140625, -0.7255859375, 0.8232421875, -0.036346435546875, 1.2021484375, -0.12017822265625, -0.5986328125, -1.28515625, -0.262451171875, -0.7421875, -0.27392578125, -0.59375, 0.392333984375, -0.72802734375, -1.0732421875, -0.54541015625, -0.273193359375, 0.94970703125, 0.20849609375, 1.26171875, -0.046783447265625, 1.083984375, 0.1568603515625, 1.484375, 0.0994873046875, -0.436767578125, -0.468994140625, 3.8125, 0.3642578125, -0.23583984375, -1.6064453125, -0.36279296875, 0.0611572265625, 0.78076171875, -1.3515625, -0.285888671875, -0.55517578125, 0.69189453125, -0.429443359375, -0.288818359375, -0.56689453125, 0.74755859375, 0.47607421875, -1.96875, -0.455078125, 0.60986328125, 3.177734375, -0.1522216796875, 1.41796875, -1.8408203125, 0.1104736328125, 1.013671875, -0.73388671875, 0.381103515625, 0.2081298828125, 2.037109375, 0.998046875, -0.5732421875, -0.52783203125, -0.75146484375, 1.2138671875, -0.485595703125, -2.3203125, 0.4140625, 0.22216796875, 0.1375732421875, -2.546875, 1.3779296875, -0.10760498046875, 0.1444091796875, 0.0246124267578125, 1.2333984375, -0.90283203125, 0.59521484375, -0.1275634765625, 0.50439453125, 1.3291015625, -1.0517578125, 1.193359375, 2.185546875, 0.393798828125, -1.5869140625, -1.4521484375, 0.31396484375, 1.189453125, -0.7880859375, 1.0712890625, -0.99853515625, -1.33984375, -2.6875, -0.2578125, -0.280029296875, 2.150390625, -1.333984375, -0.87255859375, -0.6796875, 0.34375, 1.861328125, -0.6484375, -0.1600341796875, 0.57177734375, 1.9443359375, 0.34521484375, 1.0830078125, 0.157470703125, -1.119140625, 0.623046875, 0.1192626953125, -0.76025390625, 0.303955078125, -1.2490234375, -0.416259765625, -0.285888671875, -0.1727294921875, -0.56591796875, 0.06292724609375, -0.1729736328125, -0.98095703125, 0.057525634765625, -1.2763671875, 1.234375, 0.8193359375, -0.2349853515625, -1.1279296875, 0.81591796875, -0.1475830078125, -0.80908203125, -1.708984375, -0.02508544921875, 1.1552734375, 0.8310546875, 1.0869140625, -0.84375, 0.1243896484375, -0.044219970703125, -0.6337890625, -1.5537109375, 0.1217041015625, -1.3583984375, -0.2271728515625, -0.59423828125, -1.4482421875, -0.57470703125, -0.487548828125, -0.8134765625, 1.361328125, -0.47900390625, -0.51806640625, -0.29443359375, -0.517578125, -0.6728515625, -0.450439453125, -1.8515625, -0.04345703125, 0.2105712890625, 0.92578125, -0.0718994140625, -0.5029296875, -0.8486328125, 0.07135009765625, 1.72265625, 1.3056640625, 0.68603515625, 0.6982421875, 0.8125, -0.4541015625, -0.69775390625, 1.1240234375, -1.4921875, -0.25390625, -0.18212890625, 0.93701171875, -0.2296142578125, 0.447265625, 0.3642578125, 0.06329345703125, -0.44287109375, 0.362060546875, 0.458251953125, -0.044677734375, 0.52587890625, -0.49609375, 1.5, -0.96435546875, -0.68115234375, 0.99169921875, -0.7763671875, -0.495361328125, -0.299072265625, 1.5458984375, 0.97998046875, 0.451904296875, -1.0263671875, 1.18359375, -0.057159423828125, 2.16796875, 0.062042236328125, -0.0843505859375, -1.212890625, -0.71923828125, 0.7724609375, 1.638671875, 1.3720703125, -1.8388671875, 0.673828125, 1.412109375, -0.86376953125, -0.93310546875, -0.41748046875, -0.7197265625, 0.2432861328125, 1.001953125, -0.8232421875, -1.1884765625, 0.06298828125, -0.66259765625, -1.9833984375, -0.07000732421875, 0.75927734375, 1.30078125, 0.419921875, 1.046875, -0.431396484375, -0.6328125, -0.58544921875, 1.951171875, -1.1279296875, 0.53271484375, -0.17529296875, -0.457275390625, 0.27587890625, 0.0091094970703125, 0.1849365234375, -0.498291015625, -0.396728515625, 1.4609375, 0.2252197265625, -0.4150390625, -0.58642578125, 1.2998046875, -1.2216796875, 0.7099609375, 1.4892578125, -1.3603515625, 0.46044921875, -0.84033203125, -0.3544921875, 2.7890625, 0.1591796875, 0.93017578125, 0.5966796875, -0.568359375, -1.3564453125, 0.5185546875, 0.309814453125, 0.9443359375, -1.642578125, 0.9423828125, 0.460693359375, -0.57861328125, 0.58984375, -0.29296875, -0.8349609375, -1.4375, 0.630859375, -0.34716796875, -0.76806640625, 1.94140625, -0.60107421875, 2.865234375, -0.7822265625, -0.79150390625, -0.54345703125, 1.0537109375, 0.242919921875, -1.359375, -2.380859375, 1.1005859375, 0.3388671875, -0.1341552734375, -0.304443359375, 1.328125, -1.955078125, 1.5859375, 0.83203125, -0.8779296875, 1.2421875, -1.1005859375, 1.166015625, -0.11871337890625, -0.876953125, -0.347900390625, 0.234375, -0.308837890625, 0.25244140625, -1.0166015625, 0.28955078125, 0.72314453125, -0.40478515625, -0.748046875, 1.2998046875, -0.164306640625, -1.697265625, -2.19921875, -0.08135986328125, 0.6591796875, 1.107421875, 0.1043701171875, 1.158203125, 0.9677734375, -1.8251953125, 0.904296875, -0.783203125, -0.1080322265625, -1.7705078125, -1.8681640625, -1.3935546875, 0.376953125, 0.79443359375, 1.1201171875, -0.19482421875, -1.888671875, -0.10797119140625, 0.359375, -0.65185546875, 0.4970703125, -1.34375, -1.310546875, -2.361328125, 0.479248046875, 0.06549072265625, 1.2255859375, -0.8115234375, -1.978515625, -0.1201171875, -0.410400390625, -0.1990966796875, -0.150146484375, 1.5439453125, 0.5322265625, 0.92333984375, -1.3349609375, 0.1795654296875, -0.0240325927734375, -1.2998046875, 2.1640625, 0.92919921875, 0.1070556640625, 0.3427734375, 1.37109375, 0.1400146484375, 0.341064453125, 0.1983642578125, -0.54150390625, 0.327392578125, 1.396484375, 1.0263671875, -0.8720703125, -0.326904296875, -1.234375, -1.1982421875, -0.34716796875, 0.187744140625, -1.24609375, -0.1568603515625, -0.75146484375, 0.2227783203125, -1.361328125, -0.418212890625, -1.0517578125, -0.68798828125, -2.044921875, -0.473876953125, -1.8505859375, 0.50390625, 0.1796875, 0.085693359375, -2.1015625, 1.9326171875, 0.254638671875, 0.74072265625, 1.32421875, -0.27978515625, -0.83056640625, -0.386474609375, -0.38330078125, -1.0771484375, 0.221435546875, 0.96484375, -1.24609375, 0.671875, -0.55615234375, -0.94921875, 0.6865234375, 0.51171875, 0.415283203125, -1.341796875, -1.083984375, 1.0048828125, 0.08721923828125, -0.170166015625, -1.013671875, 0.5283203125, -0.42724609375, 0.79345703125, 0.84912109375, 0.359619140625, -0.5498046875, 1.015625, 0.1480712890625, 1.158203125, 1.1064453125, -0.87255859375, 0.019622802734375, -1.5078125, 1.146484375, 2.314453125, 0.399658203125, 0.2371826171875, -1.8681640625, -0.2327880859375, 0.849609375, -0.337158203125, 1.6171875, -0.421142578125, 1.8037109375, -0.6640625, 1.2861328125, -2.4921875, -0.2086181640625, 0.84130859375, -1.2607421875, 0.172607421875, 0.95703125, 0.84521484375, -0.52099609375, 0.132568359375, -0.05413818359375, 0.343017578125, 0.1588134765625, 0.189453125, 2.361328125, -0.01326751708984375, -0.54150390625, -0.40869140625, 0.69970703125, -0.10272216796875, -0.88525390625, 2.078125, 0.276611328125, 0.8017578125, -0.0009255409240722656, 1.0244140625, -0.5732421875, -0.58544921875, 0.1302490234375, -0.98681640625, -0.156982421875, -1.078125, 0.86328125, 2.1875, -1.43359375, 0.232421875, 0.1942138671875, 0.416748046875, -0.646484375, -0.133056640625, 0.47705078125, 0.4326171875, -0.005825042724609375, 0.556640625, -1.3701171875, -0.32373046875, -0.2398681640625, 0.161376953125, -0.62158203125, -0.455322265625, -1.7431640625, -0.53271484375, 1.0400390625, 0.066650390625, -1.978515625, -0.0308837890625, 1.6796875, -1.4345703125, -1.8955078125, 0.88623046875, 1.947265625, -0.33251953125, -0.486572265625, 0.040008544921875, 0.291015625, 1.1875, 0.3984375, -0.75830078125, -1.51953125, -1.1123046875, 1.189453125, 1.2119140625, 0.07366943359375, 0.6171875, 0.61083984375, 0.062164306640625, 0.39697265625, -0.147705078125, -0.202880859375, -0.263427734375, -0.9560546875, 1.671875, -0.0181427001953125, 0.24365234375, -1.70703125, -0.391357421875, -0.9365234375, -1.2841796875, 0.32568359375, -0.278564453125, 2.818359375, 1.197265625, 0.400390625, 0.67333984375, -0.76220703125, 0.094970703125, 0.55078125, -0.58740234375, 0.6494140625, -2.27734375, -0.3544921875, -0.388671875, 0.470947265625, 0.7509765625, -0.462890625, 0.822265625, -1.2080078125, 0.53564453125, -1.0107421875, 0.0531005859375, -0.70361328125, 0.7998046875, -2.021484375, -0.38037109375, -0.48095703125, -0.66552734375, 1.142578125, -0.1519775390625, 0.58251953125, 1.421875, -0.56689453125, -0.58740234375, 0.0938720703125, 0.55810546875, 1.0263671875, 1.126953125, -0.7587890625, -0.7919921875, -0.363037109375, -0.47607421875, -0.32275390625, 0.06549072265625, -0.8388671875, -0.41748046875, -0.541015625, 0.40625, -2.404296875, -0.8154296875, 0.77783203125, -1.412109375, 1.1982421875, 0.78466796875, -0.2978515625, -2.2265625, 1.0439453125, 0.6181640625, -0.85400390625, -1.8251953125, -1.478515625, 0.480712890625, 0.468505859375, -0.0452880859375, -0.9248046875, 2.505859375, -0.50341796875, -0.7275390625, -0.057464599609375, -0.92431640625, -0.5361328125, -0.07879638671875, 0.236328125, -0.1788330078125, 0.0213470458984375, -1.6796875, -0.69873046875, -1.673828125, 0.2337646484375, 1.498046875, 0.89013671875, -0.8837890625, -0.869140625, -0.44482421875, 0.049896240234375, -0.06903076171875, 0.053680419921875, -0.2265625, 0.2279052734375, 0.424560546875, -0.2205810546875, -0.2491455078125, -0.54052734375, -1.3154296875, 0.89599609375, 1.7099609375, 0.8935546875, 0.0092315673828125, -0.81103515625, 1.29296875, 1.361328125, -0.1553955078125, 1.056640625, -1.197265625, -0.04510498046875, 0.72021484375, -0.385498046875, 0.484375, -0.0791015625, -1.0712890625, 0.381103515625, 0.38427734375, -0.4033203125, 0.44775390625, 1.037109375, 1.9736328125, -0.271240234375, 1.666015625, -0.34814453125, -0.1341552734375, -1.1416015625, -0.833984375, 1.8720703125, 0.5029296875, -1.189453125, 0.85107421875, -1.28515625, -1.4912109375, -0.48681640625, -1.076171875, -1.9931640625, 0.23046875, -1.96875, -0.1295166015625, -0.08551025390625, 0.269775390625, 1.5947265625, 0.0919189453125, -0.1790771484375, -0.056365966796875, 0.01617431640625, -0.5458984375, -2.283203125, -0.31103515625, 0.1873779296875, 0.05438232421875, -0.415283203125, 1.6181640625, 0.64208984375, -0.0157470703125, 0.76025390625, -0.06500244140625, -0.1168212890625, -0.288330078125, 1.7646484375, 0.448486328125, 0.62841796875, 0.207275390625, 0.007587432861328125, -1.2099609375, -1.099609375, 0.85693359375, -2.220703125, 0.9248046875, 0.4296875, -1.68359375, -0.385986328125, 1.2412109375, 0.80126953125, 0.367431640625, 0.404296875, -1.134765625, -0.7294921875, -1.013671875, -0.5439453125, -0.1573486328125, 0.57080078125, 0.413818359375, 0.203369140625, -0.400390625, 0.75341796875, 0.8642578125, 0.48046875, -1.171875, 0.0625, -0.2027587890625, -0.2384033203125, 0.52880859375, -1.427734375, 0.72314453125, 0.79736328125, 0.6240234375, -0.68359375, -0.297119140625, -0.68310546875, -0.013641357421875, 0.71337890625, 0.53173828125, -0.414306640625, -0.80859375, 0.3271484375, 0.355712890625, 0.62158203125, 0.88134765625, -0.212646484375, -0.499755859375, -0.7685546875, 0.317626953125, 0.88916015625, 0.9384765625, 0.3154296875, -0.10186767578125, 0.9423828125, 0.69140625, 0.23974609375, 1.1103515625, -0.6376953125, 0.11444091796875, -1.0537109375, 0.95068359375, -0.174560546875, 0.74169921875, -0.256103515625, -0.8837890625, -1.3896484375, -0.62109375, -0.0220489501953125, -0.978515625, 1.9306640625, -0.26904296875, -1.7138671875, 0.2032470703125, -0.281494140625, 1.017578125, 0.1953125, -1.1220703125, 0.488037109375, 1.068359375, -0.39501953125, -0.3681640625, 0.461181640625, -0.6748046875, 0.869140625, -0.96875, 0.8349609375, -1.873046875, 1.1162109375, -0.50341796875, -0.364501953125, -0.50048828125, -1.6279296875, 0.32568359375, 0.57470703125, -0.1778564453125, -0.461669921875, 1.0302734375, 0.283935546875, 1.134765625, 0.64990234375, 0.1429443359375, 1.14453125, 0.91748046875, -1.673828125, -1.58203125, 1.3525390625, -0.5146484375, -0.76416015625, 0.88818359375, -1.126953125, -0.332275390625, -0.368896484375, -0.485595703125, -2.078125, 0.2783203125, 0.83984375, 0.1834716796875, 0.83056640625, 0.0280303955078125, 0.98193359375, -1.330078125, -0.83203125, 0.51953125, -0.9248046875, -0.218994140625, -0.243408203125, -1.5576171875, -0.1885986328125, -0.401123046875, -0.56787109375, -0.1495361328125, 0.283203125, -0.1710205078125, -0.37646484375, -1.607421875, 1.5126953125, -0.09033203125, 1.314453125, -2.541015625, 0.7646484375, -0.3603515625, 0.64306640625, -0.66552734375, 0.427001953125, -0.1217041015625, -0.63134765625, -1.236328125, 0.87255859375, -0.07073974609375, 1.0234375, -0.74365234375, 0.322265625, -1.78125, 1.1611328125, 0.92626953125, -1.697265625, 0.61083984375, 0.09051513671875, 0.2061767578125, 0.86083984375, -1.3125, -0.10675048828125, 0.61328125, -0.142822265625, -0.285400390625, 1.994140625, -0.19189453125, -0.10382080078125, -0.081787109375, -1.48828125, -0.1064453125, 0.45947265625, 0.6201171875, -1.0625, -0.03228759765625, 1.1376953125, 1.2509765625, -0.0291748046875, -0.50048828125, -0.556640625, 1.0771484375, -1.1806640625, 0.9765625, 0.5927734375, -0.10491943359375, -0.336181640625, 0.849609375, 0.1094970703125, -0.71142578125, 0.7900390625, 0.6123046875, -0.822265625, -0.371826171875, -0.8134765625, -1.6708984375, 1.095703125, -0.93603515625, 0.73388671875, 0.65869140625, -1.689453125, -0.055572509765625, -0.460693359375, 1.1337890625, -1.0634765625, 0.09918212890625, 0.56591796875, 0.373779296875, 0.84033203125, 1.064453125, 0.157958984375, -0.051727294921875, 0.92333984375, 1.9921875, 0.30029296875, -1.357421875, 0.1346435546875, -0.64453125, 0.1605224609375, 0.142822265625, 0.20751953125, 0.98876953125, -1.4580078125, -0.93115234375, 1.1083984375, -0.09246826171875, 1.8134765625, 0.484375, -0.90283203125, -1.1630859375, -1.2392578125, 0.71923828125, -0.9404296875, 0.18359375, 1.6474609375, -0.919921875, 0.5703125, -0.158447265625, -0.041961669921875, -1.3779296875, 0.2012939453125, -0.306396484375, 0.2452392578125, 0.09588623046875, 0.22021484375, -0.168212890625, -0.72216796875, -0.331787109375, -0.66943359375, -0.60791015625, -1.3486328125, -2.140625, 0.320068359375, 0.861328125, 0.45947265625, 1.6552734375, -0.215087890625, 0.52685546875, -0.94775390625, 0.3779296875, -1.4609375, -1.2138671875, -0.1458740234375, -0.5244140625, 0.77392578125, 0.2406005859375, -1.2578125, -0.357421875, 0.525390625, 0.76171875, -0.8896484375, 1.583984375, 0.01084136962890625, -0.95654296875, -1.0390625, -0.055572509765625, 0.58154296875, 0.003849029541015625, -0.2919921875, -0.00960540771484375, -0.442138671875, -0.89501953125, 0.54833984375, -1.9677734375, 1.6220703125, -0.1346435546875, 0.237060546875, -0.57275390625, 1.2998046875, -0.304443359375, 0.09307861328125, 0.9130859375, 1.287109375, -1.220703125, 0.90185546875, 0.084716796875, 0.69482421875, -1.666015625, -0.457763671875, -0.294921875, -0.45703125, -1.048828125, 1.30078125, -0.06298828125, -1.474609375, 0.28466796875, 0.9970703125, -1.0537109375, 0.77392578125, -1.517578125, -0.59228515625, 0.010345458984375, 0.62841796875, -0.38232421875, 0.2384033203125, 1.4306640625, -0.5546875, -1.869140625, 0.6943359375, -0.300048828125, 1.0869140625, -0.5029296875, 2.21484375, 0.6279296875, -0.0513916015625, 0.49951171875, 0.4248046875, 0.6884765625, 0.43359375, 0.358642578125, -0.2303466796875, -1.5224609375, -0.414306640625, -1.21484375, 1.400390625, -0.1685791015625, 1.55859375, -0.09893798828125, 1.35546875, -1.8310546875, 1.4326171875, -0.14794921875, -0.15087890625, -0.1939697265625, 1.2578125, 1.57421875, -0.69775390625, 0.411865234375, 0.408447265625, 0.66455078125, 2.080078125, 0.7421875, -0.304931640625, -0.39404296875, -1.978515625, -0.1575927734375, 0.2132568359375, -0.24169921875, -0.1220703125, -1.7236328125, -1.1767578125, 1.8466796875, 0.7041015625, 0.5478515625, -0.423095703125, 0.2958984375, 0.4228515625, -1.3291015625, 0.96728515625, 0.4296875, 0.152587890625, 0.438720703125, -0.493896484375, -0.6484375, -1.884765625, -0.328857421875, 1.6123046875, 0.923828125, -0.75244140625, 0.41357421875, -0.74560546875, -0.80224609375, -0.474609375, -0.1888427734375, 2.951171875, -1.1005859375, -0.83349609375, -1.23046875, 0.32568359375, 0.00772857666015625, 0.7373046875, -1.255859375, 0.185546875, 0.331787109375, -0.93505859375, 0.442138671875, 0.54541015625, 1.2822265625, -0.1378173828125, -0.08935546875, 1.4560546875, 0.6142578125, 2.021484375, 1.970703125, 1.5068359375, -2.240234375, 1.2939453125, -2.5, -0.28662109375, 0.474365234375, 1.4765625, 0.2220458984375, -1.724609375, -0.7978515625, 0.3896484375, 0.8916015625, -0.9658203125, -1.02734375, -0.984375, -0.051422119140625, -0.01183319091796875, -0.36572265625, 0.2122802734375, 0.06744384765625, 0.2164306640625, -0.9560546875, -0.97265625, -1.1259765625, -0.5078125, -0.935546875, 2.328125, -0.8388671875, -1.5986328125, 0.96484375, 0.498291015625, -1.23828125, -0.41943359375, -0.443115234375, 0.2476806640625, 1.2490234375, 0.2081298828125, -1.19140625, -1.2001953125, -2.015625, -0.297119140625, 1.0400390625, 0.061431884765625, 0.67138671875, -0.27099609375, -0.6103515625, -1.0927734375, 0.669921875, -0.0784912109375, -0.419677734375, -1.2685546875, 0.3076171875, 1.53125, 0.2198486328125, 0.31689453125, -0.677734375, -2.14453125, 0.64892578125, -0.70751953125, -0.1494140625, 1.12109375, -0.49609375, 0.95166015625, 1.5703125, -0.80029296875, 2.46875, 3.322265625, 0.7861328125, -0.7197265625, -2.54296875, -1.16796875, 0.145751953125, -0.460205078125, -2.2578125, -1.0986328125, -0.39599609375, 0.49365234375, -0.931640625, 0.525390625, 1.4169921875, -0.483642578125, -1.412109375, 0.689453125, 0.3115234375, 0.21923828125, -0.283447265625, -1.078125, 1.240234375, -1.419921875, 1.650390625, 1.173828125, 0.90869140625, 1.6953125, 1.048828125, 0.2012939453125, 0.0606689453125, -0.15966796875, 0.8515625, -0.80419921875, 1.0361328125, -1.8359375, 0.06658935546875, 0.93212890625, 2.095703125, -0.38720703125, -0.08953857421875, -0.83349609375, -0.72314453125, 0.16748046875, 0.8935546875, -1.2275390625, 1.4462890625, 0.1845703125, -1.4580078125, -0.268310546875, 0.63720703125, 2.21484375, 0.81591796875, 1.251953125, 0.036376953125, 0.1256103515625, -0.445068359375, -0.67236328125, -0.1988525390625, 1.876953125, -0.2666015625, 1.2001953125, 0.78759765625, 0.58935546875, 1.736328125, -0.1422119140625, 0.215087890625, 0.488525390625, 1.1318359375, -0.6650390625, -0.28369140625, -1.3046875, 0.2354736328125, -1.791015625, -0.228515625, 1.0263671875, 0.58349609375, -0.2646484375, 0.78759765625, 0.4853515625, -0.63525390625, 1.271484375, -1.3447265625, -0.9228515625, -0.751953125, 0.412109375, 0.28564453125, 0.25830078125, -1.4599609375, 0.5322265625, 1.2939453125, -0.3740234375, 0.2484130859375, -2.529296875, 1.564453125, -0.28759765625, 0.69140625, -1.546875, -2.205078125, 0.62841796875, 0.90673828125, -0.044891357421875, -0.5087890625, -0.0159454345703125, -0.61865234375, 1.1025390625, 0.7001953125, 0.998046875, 1.103515625, 0.37353515625, -0.87890625, 1.966796875, 0.40625, 0.2421875, -0.70849609375, 1.6962890625, -1.0654296875, 1.0244140625, 0.1298828125, -1.3896484375, 1.4501953125, -1.294921875, 1.2353515625, 1.1484375, -1.8837890625, 1.57421875, -1.3115234375, -0.473876953125, -1.4228515625, -0.62890625, 1.2939453125, 0.3115234375, 0.95068359375, -2.212890625, 1.5029296875, -0.496337890625, 2.25, 0.14111328125, 1.0810546875, -0.040191650390625, -0.5244140625, -0.1888427734375, -1.3642578125, -0.44775390625, -1.60546875, 0.26123046875, 0.75927734375, 2.453125, 0.6552734375, -0.1641845703125, 1.236328125, 0.8681640625, -0.7705078125, 0.8701171875, 0.68798828125, 0.357177734375, 0.55322265625, -0.27685546875, -0.349609375, -1.029296875, -1.2412109375, -1.947265625, 2.095703125, 1.3291015625, -0.07940673828125, 0.457763671875, 0.71533203125, -0.732421875, -1.5029296875, -1.50390625, -0.040130615234375, 1.80859375, 1.068359375, 0.94091796875, -1.154296875, -1.6474609375, -0.075927734375, 1.6572265625, 1.1640625, -0.71630859375, -2.228515625, -2.30859375, -0.98388671875, 0.71435546875, -1.3466796875, -1.05859375, 1.140625, -0.00504302978515625, -1.8564453125, -0.97119140625, 0.441162109375, 0.6494140625, -0.307373046875, 0.1502685546875, -0.231689453125, 0.20263671875, -2.185546875, -0.08013916015625, -0.1441650390625, -1.0078125, 1.69140625, -1.326171875, 1.0595703125, 0.97265625, -1.1171875, 1.6240234375, -1.2275390625, -0.859375, 0.79541015625, -0.8837890625, 1.5537109375, 1.1845703125, -1.46875, -1.4111328125, -0.59912109375, 0.112060546875, 0.029052734375, -2.044921875, -1.287109375, 0.256103515625, -0.3310546875, 0.8388671875, 0.818359375, 1.3994140625, -0.716796875, 0.03497314453125, 0.376708984375, -1.5302734375, -1.3671875, 0.61865234375, 0.87548828125, -0.047119140625, -0.161376953125, 0.262939453125, -1.294921875, 0.3369140625, 0.71435546875, -0.244873046875, -0.4013671875, -0.36279296875, 1.5732421875, 0.3388671875, 0.391357421875, 0.52880859375, 0.69287109375, 0.5146484375, -2.251953125, -1.6513671875, -1.4169921875, -1.6376953125, -0.38818359375, 0.7333984375, 0.1591796875, -0.625, 0.7216796875, -0.0777587890625, -1.0068359375, -1.3173828125, -0.82666015625, -1.6845703125, 0.7412109375, 0.58837890625, -0.5908203125, -1.70703125, -0.468505859375, 0.266357421875, 0.384765625, -0.244873046875, 0.841796875, 1.927734375, -1.2802734375, -1.240234375, -0.11737060546875, 1.125, 0.2587890625, -1.8134765625, 0.9775390625, 0.64990234375, 0.6923828125, 0.8349609375, -0.82666015625, 0.267822265625, -0.381591796875, -1.375, 0.69775390625, 0.86474609375, -0.67138671875, -0.78515625, 0.1630859375, 1.1669921875, 1.07421875, 1.0693359375, -0.4833984375, -0.2734375, -0.84765625, -0.324951171875, 1.8779296875, 0.9091796875, -0.55029296875, 1.52734375, 0.06622314453125, 0.3349609375, 0.60546875, -0.496826171875, -1.708984375, -0.818359375, -0.52587890625, 1.1064453125, 1.1103515625, -1.2314453125, 0.8505859375, 1.033203125, -1.5576171875, 0.22216796875, -1.0888671875, -1.916015625, 1.123046875, 0.2239990234375, -1.8466796875, -0.032196044921875, 1.7470703125, 1.642578125, -2.072265625, 1.154296875, 2.126953125, -0.1900634765625, -0.06451416015625, 0.072265625, 2.052734375, -0.193603515625, 2.169921875, 0.630859375, 0.55078125, -2.3515625, 1.2978515625, 1.326171875, 0.277587890625, 0.0963134765625, -0.0240478515625, -1.345703125, 0.375, 0.0197601318359375, -0.43994140625, -0.26318359375, 0.86669921875, -0.381591796875, 0.1463623046875, -0.424560546875, 1.619140625, -0.830078125, -0.56982421875, -1.009765625, -1.7958984375, 1.779296875, -1.3447265625, -0.2254638671875, -0.76611328125, -0.245849609375, 2.08203125, -0.016143798828125, -1.6826171875, 0.84619140625, 0.1212158203125, -0.79931640625, 0.40771484375, -0.1944580078125, 1.03515625, 0.61865234375, 0.2244873046875, 0.050445556640625, 1.2919921875, -0.74755859375, -0.68408203125, 0.76025390625, -2.65234375, -0.274169921875, -1.2412109375, -2.55078125, 0.57666015625, -1.0849609375, -0.13525390625, 1.421875, 0.13427734375, -2.03125, 1.1650390625, -1.6748046875, 0.33935546875, 0.47900390625, 0.17529296875, 0.184326171875, -1.4228515625, 0.697265625, -0.2103271484375, -0.970703125, -1.044921875, 0.397216796875, 1.1787109375, 0.56396484375, 0.298828125, 0.94189453125, 0.66748046875, 0.00899505615234375, -1.2548828125, -0.19384765625, -0.70751953125, -1.4716796875, 1.591796875, -0.68798828125, -2.07421875, -1.1962890625, -0.89794921875, 0.5751953125, -0.319580078125, 1.142578125, -0.41748046875, -0.043365478515625, 0.51513671875, 0.2369384765625, 0.302001953125, 0.21875, 0.919921875, 2.173828125, 1.2236328125, 0.052947998046875, -0.97119140625, -1.3115234375, -0.08868408203125, 2.83203125, 1.4228515625, -0.329345703125, -0.24755859375, 0.59228515625, -0.300048828125, -0.48046875, -1.6181640625, 0.437255859375, -0.038909912109375, -1.1298828125, -0.46875, 1.185546875, -0.78173828125, -1.7724609375, -0.448486328125, -0.5966796875, 0.345947265625, 0.15966796875, 0.0166168212890625, 0.292236328125, -0.281494140625, 1.2734375, -0.82861328125, 1.7216796875, -1.2841796875, 0.388671875, 1.029296875, -0.78955078125, 0.34033203125, -0.341064453125, 0.4326171875, 0.869140625, 0.157958984375, -1.9853515625, -0.1181640625, -0.328369140625, 0.30810546875, 0.2352294921875, 1.779296875, -1.00390625, 0.90380859375, -0.1207275390625, -0.1737060546875, 0.658203125, -0.9404296875, -0.0213623046875, 0.5166015625, 0.93701171875, 1.765625, 0.2420654296875, 0.036224365234375, 0.36279296875, 0.6826171875, 2.21875, 1.1396484375, -1.47265625, -0.2022705078125, 0.3564453125, -1.51953125, 0.197998046875, 0.451416015625, 0.75634765625, 0.13720703125, -1.1142578125, -1.427734375, -0.195068359375, -0.73876953125, 1.4072265625, -0.16845703125, 1.1435546875, 0.8662109375, 0.393798828125, -0.70361328125, 0.327880859375, -0.35888671875, -0.0258331298828125, 1.0048828125, 1.375, 0.6513671875, -1.654296875, -0.455322265625, 0.3115234375, 2.419921875, -0.208251953125, 0.52978515625, -0.86865234375, -0.50244140625, -0.06134033203125, -1.623046875, -0.323486328125, 0.110595703125, 0.5390625, -1.2607421875, -0.712890625, 0.6396484375, 0.517578125, -0.2060546875, -0.859375, -1.05859375, -1.30078125, -2.517578125, -2.498046875, -1.29296875, 1.8779296875, -0.5244140625, 1.3974609375, -0.68603515625, -0.86865234375, 0.486572265625, -1.384765625, -0.76953125, 0.5673828125, -0.65185546875, -0.2073974609375, 0.229736328125, -0.440185546875, 1.7568359375, 0.10675048828125, 0.18017578125, -0.345458984375, -0.61669921875, 0.357666015625, 1.533203125, -0.32177734375, 1.21484375, 1.6552734375, -2.595703125, 1.751953125, 1.220703125, 1.123046875, 0.5185546875, -0.2025146484375, 0.7802734375, -0.283935546875, 1.4716796875, 1.1376953125, -1.5244140625, -0.80419921875, 0.970703125, 0.54638671875, -0.7431640625, -1.251953125, -0.3955078125, -2.404296875, -0.201416015625, 0.86572265625, -0.986328125, 0.280029296875, -1.4853515625, -0.1373291015625, -0.171142578125, 1.310546875, 0.66845703125, 1.2353515625, 0.08514404296875, 0.367919921875, 0.60302734375, -2.794921875, 1.056640625, -0.449462890625, -0.50341796875, -0.5693359375, 0.247314453125, 1.1171875, -0.25048828125, -0.1575927734375, 0.388671875, -0.58056640625, -0.625, 2.326171875, 1.7294921875, 0.347900390625, -1.763671875, -1.1787109375, 2.755859375, -1.0185546875, -1.4775390625, -0.873046875, 0.72265625, 0.118896484375, 0.578125, -0.70263671875, 1.1875, 0.81005859375, 0.1273193359375, 0.5048828125, 0.1961669921875, 0.09527587890625, 0.9130859375, 0.091552734375, 0.31201171875, -0.982421875, 1.400390625, 0.7314453125, 0.78271484375, 0.40234375, 0.376220703125, -0.2391357421875, 0.517578125, -1.2880859375, 0.236083984375, -1.0498046875, 0.9404296875, 0.95556640625, -0.275146484375, 0.42333984375, -1.7197265625, -0.74951171875, -0.07562255859375, 1.3662109375, 0.10931396484375, -0.85791015625, -1.3662109375, 0.9521484375, -0.474365234375, 0.3671875, -0.004344940185546875, 0.60009765625, -0.6640625, 1.1982421875, 0.90234375, 1.9970703125, 0.3291015625, 1.236328125, -1.4375, -0.1085205078125, -1.421875, -0.48828125, 1.0361328125, 1.3017578125, 0.251708984375, -0.09539794921875, 0.5078125, 1.0625, 1.1533203125, 0.8837890625, -0.3388671875, -1.724609375, 1.7685546875, -0.41015625, -0.58251953125, 0.003955841064453125, 0.9384765625, -0.40673828125, -1.2119140625, 0.208984375, 1.611328125, -0.708984375, 0.51220703125, 1.078125, -0.9365234375, -0.52978515625, 0.1988525390625, -0.1925048828125, -0.4521484375, -0.65576171875, 0.004566192626953125, 0.99951171875, -0.953125, 1.6904296875, -0.06732177734375, 0.66748046875, 0.295654296875, -0.78564453125, -0.02362060546875, -1.8056640625, 1.3505859375, -1.3701171875, 1.0029296875, 0.6435546875, -1.1416015625, 1.494140625, -0.5029296875, -0.281494140625, 0.49365234375, 1.103515625, -3.220703125, 0.9794921875, -0.278564453125, -1.4365234375, -1.5380859375, 2.740234375, -0.68603515625, 0.296142578125, -0.397216796875, -2.22265625, 1.6748046875, -0.806640625, -0.280029296875, 0.24658203125, 0.76611328125, -0.6767578125, 0.8486328125, -0.0194091796875, -0.2454833984375, 1.6044921875, 1.15234375, 0.45849609375, 0.486083984375, 0.49462890625, 1.1455078125, 0.239013671875, 1.7373046875, 0.1724853515625, 0.583984375, -0.301513671875, 0.244140625, 0.65771484375, 0.259765625, -0.447021484375, -1.7177734375, -0.114501953125, 1.6943359375, -0.36083984375, 1.8134765625, 0.27978515625, -0.29345703125, -1.046875, -1.2177734375, 1.296875, 0.314453125, 1.3828125, 0.75830078125, 1.3642578125, 0.1497802734375, -0.443359375, -0.10675048828125, -0.5537109375, -0.5703125, 0.1717529296875, -2.130859375, -0.245849609375, 0.217041015625, 0.369140625, 0.440185546875, -0.99658203125, -0.0654296875, 1.1533203125, -0.46337890625, 1.0947265625, -0.392822265625, 1.7568359375, 0.357421875, -0.6923828125, -0.1878662109375, 0.3779296875, 0.00167083740234375, 0.03778076171875, -1.00390625, 0.88525390625, -1.2236328125, -0.289794921875, -1.064453125, 0.52734375, 0.61572265625, -1.8564453125, 0.97607421875, 1.330078125, -0.6591796875, -0.328857421875, 1.123046875, 0.02178955078125, 0.3310546875, 0.943359375, 0.308837890625, -0.609375, 0.037445068359375, 0.6484375, 0.6416015625, -1.0634765625, 0.5390625, -0.352294921875, 0.2783203125, 1.5693359375, 1.25, 1.4736328125, 1.998046875, 0.33154296875, 0.53076171875, 0.159912109375, 0.260986328125, 0.98388671875, 0.61767578125, -1.263671875, -0.024078369140625, 0.4931640625, -1.6982421875, -0.578125, 0.004390716552734375, 0.06854248046875, 1.13671875, 0.5458984375, 1.111328125, 1.1416015625, 1.69921875, -0.32373046875, -0.09332275390625, -0.2998046875, 0.3388671875, -0.8291015625, 0.3212890625, 0.5009765625, -1.3291015625, -0.61962890625, 0.161865234375, 0.5966796875, -0.03753662109375, 1.078125, -1.697265625, 1.9736328125, 0.348388671875, 0.3857421875, 0.187255859375, -0.1875, -1.4130859375, -1.03125, -0.1329345703125, 1.1435546875, -0.28955078125, 0.0552978515625, 0.0345458984375, -0.94189453125, 0.420166015625, 0.52490234375, 0.39404296875, 0.7216796875, -1.3564453125, 1.5009765625, -0.8173828125, -0.58740234375, 0.293212890625, 1.1787109375, -1.2998046875, 1.044921875, -0.005870819091796875, -0.9736328125, 0.73779296875, 0.2359619140625, -0.13134765625, -0.52197265625, 0.5244140625, -0.128662109375, 2.919921875, -1.099609375, -0.156005859375, 0.72119140625, -1.912109375, 0.95556640625, -1.0712890625, -0.53515625, 0.266845703125, -0.677734375, 0.3134765625, 2.521484375, 0.1727294921875, 0.947265625, 1.1103515625, 0.088134765625, 1.1162109375, 1.4130859375, 1.529296875, -0.00780487060546875, 0.281005859375, -1.0810546875, 0.155517578125, -0.375, 0.50732421875, -0.4951171875, -0.10650634765625, 0.394287109375, -1.3994140625, -0.96923828125, 0.03973388671875, -2.28515625, 1.1455078125, -0.45263671875, -0.6328125, 0.7119140625, -0.06414794921875, -2.09765625, -1.4326171875, -0.05377197265625, 0.490966796875, 0.20556640625, 0.1610107421875, 1.6123046875, 1.4833984375, 0.83154296875, -0.833984375, 0.80419921875, -0.5068359375, 0.182861328125, -0.57861328125, -0.7412109375, -0.85205078125, -1.5546875, 0.28515625, -1.0517578125, -0.343994140625, 0.468017578125, 0.6875, -0.62744140625, -1.1767578125, -3.572265625, -1.3369140625, 0.673828125, 0.279052734375, 0.69677734375, 0.5634765625, -0.2396240234375, -2.255859375, -1.236328125, 1.4169921875, 0.61328125, 0.771484375, -0.299560546875, 0.93701171875, -0.68115234375, -1.05859375, -1.0302734375, 0.908203125, -0.05908203125, -1.248046875, -1.337890625, -1.197265625, -2.21484375, -0.452392578125, -1.587890625, -1.32421875, -1.7119140625, 1.7509765625, -1.7646484375, -1.7080078125, 0.90673828125, -0.6171875, 1.1171875, -1.4033203125, 0.58154296875, 0.8203125, -0.351318359375, -0.708984375, -1.7265625, 0.57421875, -0.796875, -0.5888671875, -0.044677734375, -0.34326171875, 2.638671875, 0.47509765625, -0.486328125, -1.1005859375, 0.5771484375, -0.73291015625, -0.76904296875, 1.951171875, 0.7265625, -0.83935546875, 0.73974609375, 0.310302734375, 0.61572265625, -0.70556640625, -0.429931640625, 0.1640625, -1.1845703125, 0.93115234375, -0.4951171875, 1.33984375, -1.2724609375, 0.99951171875, 0.1925048828125, -1.056640625, -0.06268310546875, -0.103515625, 1.3564453125, 0.283935546875, -0.2440185546875, 2.193359375, 1.0087890625, 0.2900390625, -0.56982421875, 1.2744140625, 0.6845703125, 0.387939453125, -0.0782470703125, -0.86181640625, -1.8154296875, 1.1904296875, -2.21875, -1.08203125, 0.79736328125, -0.423828125, -0.2413330078125, 0.55615234375, -0.93603515625, -0.44189453125, 0.87890625, -0.462890625, -0.30322265625, 1.67578125, -0.87451171875, -2.908203125, -1.412109375, 0.258056640625, 0.20947265625, -0.90234375, -1.52734375, -0.9189453125, 0.3388671875, -1.62890625, -0.51416015625, -1.0322265625, -0.007568359375, 1.4423828125, -0.0819091796875, 0.035400390625, 0.83056640625, 1.576171875, 1.931640625, 1.2783203125, -0.7001953125, -2.513671875, 0.09539794921875, -0.72021484375, -0.087158203125, 0.1514892578125, 0.1962890625, 0.5048828125, 1.4775390625, -0.291259765625, 0.56591796875, -2.05859375, -1.0712890625, -1.650390625, -0.04742431640625, 0.8701171875, -1.65234375, -0.80322265625, -1.5126953125, 0.8955078125, -0.0665283203125, -0.9306640625, 0.72802734375, 0.9482421875, 0.1429443359375, 0.6015625, 0.69921875, 0.2281494140625, 0.1265869140625, 0.38330078125, 0.8740234375, -0.7626953125, -0.26416015625, -1.8662109375, -0.030029296875, -1.33203125, 2.51953125, 1.1025390625, -0.01190948486328125, 0.763671875, -0.031341552734375, 0.11737060546875, 0.307373046875, 2.32421875, -0.2188720703125, 1.41796875, 0.1483154296875, 0.634765625, 2.271484375, 0.288330078125, -0.355224609375, -0.2166748046875, 1.5859375, -0.31787109375, 0.217041015625, 0.51123046875, 2.025390625, 0.11383056640625, 0.00656890869140625, 0.626953125, 1.2958984375, -0.89892578125, 0.3369140625, -0.654296875, -0.0731201171875, 0.9951171875, 0.59130859375, 1.908203125, -0.88330078125, 0.5634765625, -0.58447265625, -1.7587890625, 0.1341552734375, -0.0789794921875, -1.5849609375, -0.031402587890625, -0.410888671875, 1.9501953125, -0.08270263671875, 0.21337890625, 0.050445556640625, -1.5498046875, -0.08563232421875, 0.81884765625, -1.02734375, -0.37451171875, -1.3994140625, -0.998046875, -1.1669921875, -0.2176513671875, -1.4130859375, -0.5322265625, -0.13037109375, 0.458740234375, 0.73046875, 0.473876953125, -0.78955078125, 1.3466796875, 0.73388671875, 0.962890625, -0.80029296875, -0.183837890625, 1.61328125, 1.240234375, 0.70068359375, 0.33056640625, -0.8935546875, -0.08099365234375, 0.311767578125, -0.56494140625, 1.5400390625, -0.1962890625, 1.1904296875, 0.87451171875, -1.259765625, -1.3017578125, 0.296875, 0.80615234375, -0.313232421875, 1.3857421875, -2.357421875, 0.8154296875, -1.9638671875, 0.712890625, 1.0595703125, -0.76318359375, 1.1533203125, -0.87255859375, 0.73974609375, -0.75537109375, 0.3759765625, 0.08929443359375, 0.1976318359375, 0.86865234375, -0.53076171875, -1.072265625, -1.234375, 1.44140625, 0.61767578125, -0.83251953125, -1.8623046875, 0.50634765625, 1.181640625, 0.29150390625, 0.39892578125, 1.0185546875, -3.0546875, -1.173828125, 0.5703125, 0.79345703125, -0.0677490234375, 0.359130859375, 1.189453125, 0.990234375, -2.19921875, -0.44384765625, -0.0021686553955078125, 0.75244140625, -0.1646728515625, 0.358154296875, -0.345458984375, -0.03106689453125, 0.4580078125, 0.325439453125, 1.0908203125, 0.48193359375, -0.8125, -1.021484375, -1.193359375, 0.3740234375, 0.0149078369140625, 0.037353515625, -0.31640625, -1.09375, -0.1427001953125, -0.07501220703125, 0.6845703125, 0.1671142578125, -0.1153564453125, -0.8896484375, 0.1866455078125, -0.951171875, 0.204345703125, -1.0, -0.382568359375, -0.65576171875, -0.4521484375, 0.8955078125, -1.056640625, 0.1070556640625, 0.1822509765625, -1.1806640625, 0.305419921875, -0.1422119140625, 0.08734130859375, -0.5166015625, 0.09381103515625, 2.333984375, -0.0631103515625, 0.65234375, 0.61328125, -1.416015625, 1.08984375, -0.32763671875, -0.7626953125, -2.578125, -0.2037353515625, 0.52734375, -0.07635498046875, 1.736328125, 1.5234375, 0.57275390625, 0.93896484375, 0.5537109375, -0.2186279296875, -0.318359375, -0.71435546875, 0.288330078125, -0.279052734375, -0.82275390625, -1.4248046875, -1.4541015625, 1.7919921875, 1.220703125, -1.0771484375, 1.33203125, 0.6669921875, 0.28076171875, 2.21484375, -0.451171875, 0.234130859375, -0.8330078125, -0.69189453125, 0.3876953125, 1.9326171875, -0.90380859375, 1.0556640625, -1.0595703125, -1.0771484375, 1.08984375, 1.8603515625, 0.9501953125, 0.90869140625, 0.29248046875, 1.1904296875, 0.65478515625, -1.220703125, 2.205078125, -0.48583984375, -0.6357421875, -2.19921875, -0.42822265625, 0.91943359375, 1.1142578125, -0.93798828125, -0.59228515625, 0.8515625, 1.48828125, 0.05322265625, 0.2783203125, -0.79345703125, 0.4306640625, -0.9677734375, 0.480224609375, 0.0858154296875, 0.525390625, -0.6416015625, 1.3916015625, -0.48779296875, -0.22998046875, -0.53564453125, 1.427734375, -2.140625, -0.619140625, 0.1312255859375, 1.7822265625, 0.035430908203125, 0.8818359375, -0.309814453125, 0.796875, 0.224365234375, -0.77001953125, 0.97607421875, -0.6162109375, -0.323486328125, -0.75830078125, -0.1300048828125, -0.72021484375, -1.10546875, 1.6865234375, -0.66455078125, -0.541015625, -1.8359375, -0.3095703125, -1.7861328125, 0.395263671875, -0.701171875, -1.8486328125, 0.474365234375, -1.1962890625, 0.463134765625, 0.66943359375, -1.6435546875, 0.232421875, -0.433349609375, 0.615234375, 0.51953125, -0.062469482421875, -0.128662109375, 0.203125, -0.705078125, -1.00390625, 0.127197265625, -0.85302734375, 0.421630859375, -0.771484375, 0.66845703125, 0.483642578125, -0.0033168792724609375, 0.294921875, -0.85009765625, 0.57861328125, -1.0654296875, -0.76904296875, -1.6787109375, 1.2109375, -0.71484375, -0.05975341796875, 0.72900390625, -0.6865234375, 0.35107421875, -0.4404296875, 1.5048828125, 0.93603515625, 0.0210418701171875, 0.165283203125, 1.4267578125, -0.80078125, 1.197265625, -1.486328125, -1.26171875, 0.7333984375, -0.96826171875, -1.130859375, 1.447265625, -0.171630859375, -0.94140625, 1.107421875, -0.019134521484375, 0.033294677734375, -0.60107421875, 0.94189453125, -1.1982421875, -1.580078125, 0.53125, -0.0465087890625, -0.15185546875, 1.4404296875, 1.20703125, -0.27001953125, -0.970703125, -0.1982421875, -0.2412109375, 0.8154296875, 0.2080078125, 1.75390625, -0.0172271728515625, -0.23095703125, -1.2099609375, 2.49609375, 0.0228271484375, 0.392333984375, -0.1629638671875, -0.10430908203125, -0.60107421875, -0.62548828125, -1.8291015625, -1.5703125, 2.162109375, 1.6533203125, 0.9150390625, -0.0227813720703125, 0.22265625, -1.369140625, -0.9580078125, 0.266845703125, 0.85986328125, -0.59814453125, 0.8779296875, -0.55712890625, 0.7646484375, 1.45703125, 0.8740234375, 0.1405029296875, -1.5546875, -1.837890625, 0.371337890625, 0.96044921875, -1.4814453125, 1.3447265625, -0.84521484375, -0.2265625, 1.0458984375, -0.66552734375, -0.446533203125, 1.60546875, 1.228515625, -0.498046875, -0.08026123046875, -0.83935546875, -0.87890625, -0.343017578125, 1.2431640625, -0.19189453125, -1.0244140625, 1.6865234375, 0.61865234375, 0.47216796875, 0.36572265625, 0.498046875, -0.31982421875, -0.3798828125, -0.021240234375, -0.6806640625, -0.3193359375, 1.0712890625, 0.29638671875, 0.455078125, -0.1795654296875, 1.6337890625, 0.708984375, -0.9306640625, 1.9267578125, -0.8203125, -0.70458984375, -1.3671875, -0.3623046875, 0.38330078125, 1.2646484375, -0.486083984375, -0.053253173828125, 2.064453125, -0.24951171875, 0.34912109375, -0.2587890625, 0.99853515625, 2.337890625, -0.0819091796875, -0.06793212890625, -0.024169921875, -1.2705078125, -2.021484375, 0.775390625, 0.048187255859375, -0.447998046875, -0.1678466796875, 0.1719970703125, 0.283203125, 1.5380859375, 1.6796875, -1.4609375, 0.39990234375, 1.0703125, -0.1002197265625, -0.411376953125, 0.0947265625, -0.0185089111328125, 0.038604736328125, -1.16015625, -0.44970703125, 1.533203125, 0.3125, -0.830078125, -1.8193359375, -1.7578125, -0.388671875, -0.6240234375, -2.099609375, -0.79296875, -3.263671875, -0.30029296875, 3.04296875, 2.203125, -0.324951171875, -0.49560546875, -1.12109375, 2.029296875, -0.98583984375, 1.87109375, -0.50439453125, 0.11541748046875, 0.7841796875, 0.96875, -0.88134765625, -0.0296478271484375, 1.201171875, -1.53125, 0.00043582916259765625, 1.1201171875, -0.66552734375, 0.0355224609375, -1.1630859375, -2.7265625, -0.2113037109375, 0.25634765625, -0.8681640625, 1.3154296875, 1.376953125, -0.1114501953125, 1.57421875, 0.58740234375, 0.58154296875, 1.2890625, 0.5625, 0.1881103515625, 1.931640625, -1.61328125, 0.642578125, 0.537109375, -0.578125, -2.3359375, 0.295654296875, -0.35693359375, -0.93408203125, -1.8115234375, 2.3515625, -1.478515625, 0.71142578125, 0.33056640625, -0.2939453125, 0.90283203125, -0.775390625, 0.80224609375, -1.1533203125, 1.0234375, 2.017578125, 1.7255859375, 1.5244140625, -0.1744384765625, -1.162109375, -0.107177734375, -1.0380859375, 1.3955078125, 2.216796875, 0.52294921875, 0.433349609375, 1.3798828125, -1.02734375, 0.8876953125, -0.79931640625, 0.87255859375, 0.154541015625, 0.5361328125, 0.69921875, -0.5625, 0.308837890625, -1.58203125, -0.34619140625, -1.96484375, -0.27880859375, -0.44775390625, -1.4775390625, -0.94091796875, -2.15234375, 0.578125, 1.1005859375, 1.087890625, -0.57373046875, -0.427490234375, -0.662109375, -2.486328125, -0.328857421875, -2.005859375, -0.266845703125, -1.7060546875, -0.83642578125, 1.515625, -1.923828125, 0.78662109375, 0.599609375, -2.05859375, 0.334228515625, -1.1689453125, 1.048828125, 0.0178070068359375, 0.262939453125, -0.24755859375, -0.5830078125, 0.06085205078125, 2.771484375, -1.4599609375, 0.65673828125, 0.10736083984375, -2.67578125, -0.2164306640625, 0.97705078125, 2.26953125, 1.451171875, 0.1396484375, 0.093505859375, -1.05078125, -0.039886474609375, -0.2015380859375, 0.07720947265625, -1.4443359375, -1.2392578125, 0.08306884765625, 1.123046875, -1.25390625, -0.65234375, 0.05596923828125, -1.2138671875, 0.26025390625, 1.1708984375, -0.8388671875, -0.2340087890625, 1.05078125, -0.62109375, -1.1318359375, 0.82421875, -0.68310546875, -0.05352783203125, -3.13671875, 0.0880126953125, -0.12646484375, -0.3466796875, -2.521484375, 1.7080078125, -0.517578125, -1.87890625, -1.5126953125, -0.9052734375, -0.015594482421875, -0.433349609375, 0.421142578125, -0.459228515625, 1.275390625, 2.681640625, 1.2890625, -0.79833984375, -1.3740234375, 0.1842041015625, -0.5478515625, -0.84228515625, 1.623046875, -1.0205078125, 1.2197265625, -1.787109375, -0.8212890625, -1.66015625, 0.54736328125, -2.66796875, 0.9287109375, 1.404296875, -0.95703125, 0.046875, -0.1534423828125, 0.07513427734375, 0.796875, 0.460693359375, -1.37109375, 1.01953125, 1.6123046875, -1.0947265625, 3.505859375, -0.92236328125, 0.16162109375, -0.67529296875, 0.71533203125, 1.44140625, -1.1416015625, 1.23828125, 0.279052734375, 0.61767578125, 1.3203125, -0.896484375, 0.35693359375, -0.75634765625, 0.6025390625, 0.4853515625, 0.0172882080078125, 1.5947265625, 0.53125, -1.7890625, -0.141357421875, -1.1845703125, -0.96044921875, 1.8798828125, -1.1552734375, -2.005859375, 0.90869140625, -1.154296875, 0.1568603515625, -0.1905517578125, -0.736328125, -1.021484375, -0.390625, -0.483642578125, 0.357666015625, -0.36572265625, 0.01255035400390625, 0.81298828125, -1.083984375, 0.483642578125, -0.6806640625, -0.360107421875, -0.240234375, -0.7451171875, 0.1817626953125, 0.024322509765625, -0.23681640625, 0.95947265625, -0.712890625, 0.564453125, -0.429931640625, -0.59521484375, -0.77685546875, 1.423828125, -0.055084228515625, -0.308837890625, -0.274658203125, -1.02734375, -0.337890625, 1.06640625, 0.59716796875, -1.7421875, 0.44482421875, -0.020965576171875, -1.12890625, -0.37939453125, 2.177734375, -1.2744140625, -0.93701171875, 0.2393798828125, -0.97900390625, 0.23828125, -0.1624755859375, 1.263671875, -0.360107421875, 0.5517578125, 0.27880859375, -0.0077667236328125, -0.810546875, 0.0188751220703125, -0.791015625, 2.287109375, -0.84423828125, 0.224365234375, 1.40234375, 2.029296875, -1.4326171875, -0.970703125, 0.309814453125, -0.430419921875, -1.3818359375, 0.32373046875, -2.423828125, -1.51953125, -0.09417724609375, -0.2144775390625, -0.0137481689453125, -0.46630859375, -1.5576171875, -1.501953125, 0.81005859375, -1.0009765625, -0.90576171875, -1.03515625, -1.09765625, 0.157958984375, 0.46337890625, 1.4169921875, -1.73046875, -1.4248046875, -2.39453125, -0.6396484375, 0.09075927734375, 0.35693359375, -1.626953125, 0.70068359375, 1.2490234375, 0.25732421875, 1.1884765625, -0.669921875, 0.54296875, -1.189453125, 1.6904296875, 0.51025390625, -1.4921875, 0.3212890625, 1.1015625, -0.6220703125, 1.115234375, -1.037109375, -0.64501953125, 0.9541015625, 1.67578125, -0.955078125, -1.568359375, -0.0094451904296875, -0.0034999847412109375, -0.69384765625, 0.304931640625, 1.8310546875, -0.63330078125, 0.73046875, -0.52001953125, 0.1773681640625, 0.01080322265625, -0.88623046875, 1.115234375, -0.103515625, 0.71044921875, -1.5068359375, -0.340576171875, -1.130859375, -1.4736328125, 0.2266845703125, 1.662109375, 1.4521484375, -1.33984375, -1.875, -0.27587890625, 1.4365234375, 1.3046875, -0.447998046875, 0.125244140625, -0.8662109375, -0.61328125, 1.4287109375, -1.228515625, -0.994140625, -0.896484375, -0.8779296875, -0.54345703125, -1.5029296875, -0.70361328125, -0.96142578125, -0.716796875, 0.64404296875, 0.210205078125, -1.4326171875, 0.72705078125, -0.62060546875, -2.123046875, -0.31298828125, 0.2120361328125, -2.169921875, 0.1429443359375, -0.043853759765625, -2.103515625, 0.66748046875, 0.28466796875, 0.4267578125, -0.1893310546875, 0.349609375, 0.78857421875, -0.99609375, 0.8193359375, -2.74609375, -1.720703125, -1.61328125, 1.0322265625, -2.365234375, 0.0587158203125, -0.58154296875, 0.81591796875, -1.2900390625, -0.97021484375, -1.1904296875, 0.197998046875, -1.7509765625, -0.390869140625, 0.68359375, -0.486328125, -0.37646484375, -0.423095703125, -0.587890625, -1.67578125, -0.8466796875, 0.428955078125, -0.0211029052734375, -0.6279296875, -2.212890625, -0.34326171875, 0.2310791015625, -0.453857421875, 0.98046875, 0.03778076171875, -0.333984375, 1.03515625, 0.01218414306640625, -0.58984375, -0.3125, 0.583984375, 0.246826171875, -1.7958984375, 1.5126953125, 0.6796875, 0.5341796875, 0.041961669921875, -1.6728515625, -1.521484375, -0.348388671875, 0.650390625, 1.142578125, 0.326416015625, -0.01511383056640625, 0.162353515625, -0.88916015625, 0.364501953125, 0.1981201171875, 1.4423828125, -1.2998046875, 0.6220703125, 0.26806640625, 0.379638671875, -0.331787109375, 0.321533203125, 1.6171875, 0.08795166015625, 1.5478515625, 2.1640625, -0.2978515625, 1.0078125, -0.319580078125, -2.056640625, 0.0731201171875, 0.611328125, 0.51513671875, -0.2265625, 0.3994140625, -0.94580078125, -0.260498046875, 0.037261962890625, 0.56103515625, 0.2064208984375, -1.265625, -1.13671875, 0.372802734375, 0.0080718994140625, -0.72216796875, 0.057037353515625, -1.4755859375, -1.4921875, -0.857421875, -1.796875, -2.71484375, 0.456787109375, -2.2578125, -0.18310546875, 1.166015625, -1.048828125, -0.45068359375, 0.85400390625, -1.2939453125, 0.0251007080078125, -0.2117919921875, 0.0021209716796875, 0.544921875, 0.2587890625, 0.06365966796875, -0.6875, -0.59326171875, 1.0595703125, 1.2353515625, 0.01470184326171875, -0.46728515625, 0.346435546875, -0.009185791015625, 0.947265625, 0.393798828125, -0.68701171875, 2.08984375, 1.4580078125, -0.402099609375, 1.94921875, 1.3798828125, 0.1842041015625, -1.0732421875, 0.51220703125, 0.67724609375, -0.246826171875, -1.5576171875, -1.390625, 0.58203125, -1.33984375, -2.0546875, 0.43212890625, -1.5087890625, 0.9140625, -1.0458984375, 1.9912109375, -1.33984375, 0.7607421875, -0.294677734375, 1.3271484375, -0.88623046875, -0.7744140625, 0.326171875, -1.21484375, 0.2332763671875, 1.5751953125, -1.1923828125, -1.71484375, -0.6201171875, 0.7900390625, 0.64306640625, 0.4375, -0.89404296875, -0.31005859375, -0.0063018798828125, -0.9228515625, 1.1962890625, 0.427490234375, -0.1663818359375, 1.349609375, 0.84814453125, -0.63232421875, -0.06549072265625, 0.4912109375, 2.9765625, -0.23681640625, 0.247802734375, -0.1510009765625, 1.201171875, 0.404296875, -0.056243896484375, 0.91455078125, 1.5263671875, 0.169921875, 0.89208984375, 0.10931396484375, 1.3173828125, -1.3701171875, -0.97998046875, -0.2025146484375, -0.405029296875, -1.33203125, -0.728515625, -1.0498046875, -1.8603515625, -1.7275390625, 0.78076171875, -0.2440185546875, -1.7421875, 0.447998046875, 1.564453125, 1.314453125, -0.63818359375, -0.7939453125, -1.888671875, 0.923828125, -0.580078125, -0.0027923583984375, 0.55615234375, 0.433349609375, 1.25, 0.014373779296875, -0.456298828125, -0.62841796875, 0.81591796875, 0.53857421875, 0.666015625, 0.50146484375, 1.83203125, -1.48046875, 0.1661376953125, -1.390625, 2.14453125, 0.9482421875, -1.49609375, 0.55322265625, -0.96630859375, 1.271484375, -1.716796875, -1.1328125, 0.96435546875, 1.181640625, -0.131591796875, -0.0075836181640625, 0.045867919921875, -0.01119232177734375, -1.0419921875, -0.2022705078125, -1.013671875, -0.03277587890625, 0.8798828125, 0.52392578125, 1.466796875, -0.5380859375, 0.68408203125, -0.7421875, 0.39111328125, -0.270263671875, -1.6943359375, -1.7734375, 0.47119140625, -1.083984375, -1.400390625, 0.376953125, 0.6611328125, 0.677734375, -1.2431640625, 2.69921875, 1.1064453125, 1.2216796875, 0.337158203125, 0.2193603515625, -0.2454833984375, -0.56640625, -0.5087890625, -0.337646484375, 0.0238037109375, -1.29296875, -0.60693359375, 0.1458740234375, 0.97412109375, 0.1336669921875, -0.36474609375, -0.395751953125, -0.4765625, 1.4755859375, -0.379638671875, -0.00617218017578125, -0.95361328125, 0.865234375, -1.701171875, -1.03515625, 1.8271484375, -0.357421875, 0.80517578125, -0.3759765625, 0.0982666015625, 0.958984375, 0.666015625, -0.60400390625, -0.87353515625, 0.3388671875, 1.7138671875, -0.65283203125, -0.38818359375, -2.16015625, 0.29931640625, 1.30078125, -0.5654296875, -0.2398681640625, -0.72021484375, -0.91064453125, -0.69091796875, -1.3505859375, 0.261962890625, -0.71484375, 0.82177734375, -2.2265625, 0.1781005859375, -0.496826171875, -0.79345703125, -1.5732421875, -0.08563232421875, 0.87451171875, 0.481689453125, 0.1083984375, -0.56640625, 1.255859375, 0.2022705078125, -1.583984375, -0.146728515625, 1.8046875, 0.97607421875, -0.88427734375, 0.76513671875, -1.3095703125, 0.214599609375, -1.4638671875, 0.74609375, 0.1322021484375, -1.1484375, -1.455078125, -1.826171875, -0.6328125, 0.64404296875, -1.0517578125, 0.250732421875, 0.159912109375, -1.33203125, -0.7119140625, 1.123046875, -1.7822265625, -0.7705078125, 0.044036865234375, 0.1553955078125, 0.59521484375, -0.5810546875, 0.8017578125, -0.70703125, 0.016754150390625, 1.7431640625, -0.69775390625, 0.94482421875, 0.1417236328125, -0.311767578125, -0.2012939453125, -0.66845703125, 0.88916015625, 0.1163330078125, -0.56982421875, -0.859375, 0.146728515625, -1.037109375, 0.474365234375, 0.2276611328125, 0.189208984375, 1.0556640625, -1.771484375, 0.039031982421875, 1.1123046875, 1.71484375, -0.703125, 0.62646484375, -2.22265625, 0.90625, -0.26611328125, 0.008758544921875, 1.7421875, 1.03125, 0.2056884765625, -1.326171875, 0.433349609375, -0.849609375, 1.2646484375, -0.52294921875, -0.462158203125, -1.494140625, -0.169677734375, -1.79296875, -2.654296875, -0.666015625, -1.375, 2.71875, -0.2249755859375, -0.77880859375, -0.90380859375, -0.01531219482421875, 0.283447265625, 1.501953125, 0.10302734375, -0.935546875, -0.404541015625, 0.95458984375, -0.677734375, 0.5400390625, -0.036376953125, -1.9697265625, -1.7236328125, -1.5439453125, 0.326904296875, -0.463134765625, -2.3125, -0.06439208984375, -0.84033203125, -0.434326171875, 0.460693359375, -1.4248046875, -0.4833984375, 0.76904296875, 1.708984375, -0.61669921875, -0.2200927734375, 0.41064453125, -0.92333984375, 0.55224609375, 1.9501953125, -0.33154296875, -0.343017578125, -0.364501953125, 1.01953125, 0.452392578125, 0.60400390625, 0.383056640625, 0.2042236328125, -0.89794921875, -0.89453125, 0.87744140625, 0.55859375, 1.314453125, -0.6748046875, -0.12213134765625, -1.810546875, -0.364013671875, 0.275634765625, 0.156982421875, -0.255615234375, 1.2890625, -0.67724609375, -1.3203125, 0.720703125, 2.38671875, -2.072265625, -0.9677734375, -0.0139617919921875, -0.60546875, -0.408935546875, 0.6767578125, 1.5556640625, 0.81103515625, -0.68798828125, 0.83203125, -1.443359375, 0.78271484375, 0.7470703125, -1.001953125, -1.2822265625, -1.3330078125, -0.2410888671875, -0.355712890625, 1.8759765625, 0.108154296875, -2.1328125, -2.466796875, -0.444091796875, 0.94384765625, 1.1611328125, 0.1270751953125, -0.377197265625, -0.58935546875, 0.006381988525390625, 1.740234375, 1.0283203125, 0.258544921875, 0.9404296875, -0.705078125, 0.7197265625, 0.017242431640625, -0.7109375, -0.2841796875, -0.154541015625, 0.423828125, -0.058929443359375, -0.53662109375, -0.91015625, 1.0068359375, 1.720703125, -1.46875, -0.477783203125, -0.98583984375, -0.079345703125, 0.6328125, 0.87451171875, -0.79150390625, 0.599609375, 1.162109375, -0.70751953125, -0.035491943359375, 0.295166015625, -0.55615234375, 0.128173828125, 0.299072265625, 1.00390625, -0.2587890625, 0.27392578125, 0.65869140625, -0.098876953125, 0.341064453125, -1.0341796875, 0.73779296875, -0.439453125, 0.203857421875, -0.375244140625, 0.76220703125, 1.2451171875, 0.7587890625, -0.9677734375, -0.94921875, 1.4052734375, 0.284912109375, -1.744140625, 0.2440185546875, 1.6943359375, 0.3388671875, -1.896484375, 1.74609375, -0.53271484375, 0.59912109375, -0.8955078125, 0.72802734375, -0.625, 1.017578125, 0.0162811279296875, -0.5537109375, -0.5322265625, -2.380859375, -0.431396484375, -1.759765625, -0.034423828125, 1.1845703125, 0.591796875, -0.86279296875, -0.3173828125, 0.041595458984375, -1.0712890625, 0.62060546875, -1.4033203125, -0.0589599609375, 0.469482421875, 0.1573486328125, 2.48046875, -0.402587890625, 2.453125, 1.69921875, 0.10546875, -1.09765625, 0.0753173828125, -2.19921875, -0.3701171875, -0.7392578125, -2.083984375, -0.86669921875, -0.05865478515625, -1.1669921875, 0.2025146484375, 0.83203125, -0.61865234375, 1.9765625, -0.54833984375, -0.3857421875, -0.151611328125, 2.32421875, 0.3525390625, 0.219482421875, -0.7890625, 1.1357421875, 0.58544921875, -0.1014404296875, 0.6689453125, 1.2822265625, 0.42431640625, -0.286376953125, 0.017669677734375, -0.7568359375, -0.74951171875, 0.80126953125, 1.09765625, 0.5888671875, 0.6611328125, -0.279296875, 0.57861328125, 1.15625, 1.453125, 1.1396484375, 0.10888671875, 0.312255859375, 0.572265625, -0.1207275390625, 2.212890625, 0.67822265625, -0.0428466796875, -3.533203125, 1.267578125, -0.0338134765625, -0.6025390625, 0.98876953125, 0.1416015625, 0.0382080078125, -0.9228515625, 0.390625, -0.2332763671875, -0.481201171875, 0.1771240234375, -1.0791015625, 1.84375, -0.496826171875, -0.20849609375, 0.8291015625, 0.8359375, 0.5703125, 0.66162109375, 0.189697265625, 0.8076171875, 0.418701171875, -1.09765625, 0.5380859375, 0.1002197265625, -0.94287109375, -0.269775390625, 0.568359375, 1.2314453125, -1.443359375, 0.036102294921875, -0.4716796875, 0.3984375, 0.88671875, -1.791015625, -0.54931640625, -0.352294921875, 0.31884765625, -1.236328125, -0.227783203125, -0.7197265625, 0.98828125, -2.166015625, 0.361328125, -0.19921875, 0.53369140625, 1.6337890625, 1.09375, 1.2421875, -0.433837890625, -1.4873046875, -1.578125, 1.17578125, -0.77001953125, -0.72265625, -2.0234375, -0.2166748046875, -1.24609375, -1.5615234375, -1.2060546875, 0.0003001689910888672, -0.19970703125, -0.29638671875, 1.8876953125, -0.80078125, 0.4873046875, 2.203125, -1.2822265625, -2.71875, 1.228515625, 1.3671875, -0.07257080078125, -0.0034694671630859375, -0.52978515625, -0.187255859375, -0.71630859375, 1.0947265625, 0.26953125, -0.24658203125, 1.89453125, 0.89892578125, 0.2015380859375, -1.2080078125, -0.19580078125, -0.27197265625, -0.370361328125, -1.380859375, 1.9150390625, -0.64892578125, -0.200439453125, 0.361572265625, 1.1337890625, -2.087890625, -0.41650390625, 0.7880859375, -0.43408203125, -0.41162109375, 0.91015625, 0.413818359375, 1.076171875, 0.59814453125, 1.9365234375, 0.310302734375, 1.984375, 0.381103515625, 0.304443359375, -0.0171966552734375, 0.642578125, 1.59375, -0.24609375, 0.7080078125, 1.3701171875, -1.3564453125, -0.6591796875, 0.34130859375, 0.8154296875, -1.71484375, 0.90869140625, -0.61376953125, -0.362060546875, -0.158203125, 0.8349609375, 0.1466064453125, -1.0625, 0.294677734375, -0.3916015625, 0.73193359375, 0.2454833984375, -0.8427734375, 1.826171875, -0.72802734375, -1.9052734375, 1.34765625, -0.0936279296875, -0.448486328125, -1.2373046875, 1.4658203125, 0.607421875, 1.142578125, 0.09466552734375, -0.14794921875, -0.463623046875, 1.1181640625, 0.82666015625, 0.047637939453125, -2.455078125, 0.89208984375, -0.54638671875, 0.1436767578125, 0.36181640625, 1.923828125, -0.68115234375, 0.48095703125, 0.120849609375, -0.2861328125, 0.59228515625, -1.40625, 1.4453125, -0.5078125, -1.5048828125, -1.3623046875, 0.2437744140625, -0.109375, -0.90576171875, 1.5791015625, -0.1060791015625, -0.55712890625, 0.970703125, 0.1923828125, -0.84423828125, 0.041290283203125, 1.095703125, -0.263916015625, -0.0075836181640625, -1.673828125, -0.295166015625, -0.54931640625, 2.099609375, 0.12091064453125, -1.0244140625, -0.63818359375, -0.464599609375, -0.047882080078125, -0.0498046875, 1.0703125, -0.340576171875, -0.77099609375, 0.281494140625, 0.775390625, 0.99267578125, -0.19677734375, 0.681640625, 0.37744140625, -0.37060546875, -3.2734375, -1.146484375, -0.470703125, 0.441650390625, -1.1806640625, 0.398681640625, -1.189453125, -0.7255859375, -0.11358642578125, 1.38671875, -0.349609375, 0.046295166015625, -0.393798828125, 0.75927734375, -0.127685546875, -0.1527099609375, 0.80029296875, -0.89404296875, 1.7021484375, 0.939453125, -0.6259765625, -0.52392578125, 2.15625, -0.5, 1.041015625, -1.19140625, 1.2626953125, -0.51318359375, 0.607421875, 1.0546875, 0.90625, 0.337158203125, 0.83251953125, 0.1893310546875, 2.095703125, 0.492431640625, -0.242919921875, 0.1802978515625, -0.250732421875, 0.142822265625, -1.4775390625, -0.79296875, 0.260498046875, -1.841796875, 1.6796875, -1.1455078125, -1.267578125, -0.6845703125, -0.30908203125, -0.37353515625, -1.44140625, 0.0782470703125, -0.61962890625, -0.66650390625, -0.287841796875, -0.6982421875, -1.5712890625, -1.3935546875, 0.420654296875, -1.798828125, 0.1756591796875, 0.81494140625, -1.0966796875, 0.38818359375, -1.064453125, 2.115234375, -1.521484375, -0.373779296875, -0.8955078125, -0.185791015625, -0.576171875, 0.07537841796875, 1.646484375, -1.35546875, 0.204833984375, -1.3388671875, 0.9521484375, 3.212890625, -0.51904296875, 1.275390625, -1.66015625, 0.12548828125, -0.845703125, 1.328125, 0.1697998046875, 0.41357421875, -0.1263427734375, -1.03125, -0.2027587890625, 0.45361328125, 0.74951171875, -0.052520751953125, 0.90185546875, 1.2236328125, -0.7724609375, 0.0638427734375, 1.3837890625, 1.02734375, 0.166259765625, 1.390625, -0.10400390625, -0.72998046875, 2.052734375, -0.07867431640625, -0.76806640625, 1.1064453125, 0.560546875, -0.46875, 1.828125, -0.8642578125, 0.7021484375, -0.432373046875, -0.0987548828125, 2.189453125, -1.2275390625, 0.0103912353515625, 0.62109375, -0.5380859375, -0.56689453125, 0.08282470703125, 0.556640625, -0.28271484375, -0.66259765625, 1.3017578125, -0.05194091796875, -0.6875, 0.28271484375, 0.10333251953125, 0.399658203125, 0.1539306640625, 0.028564453125, 1.55859375, -1.3681640625, -0.28271484375, -1.1630859375, -0.0292205810546875, -0.307861328125, -0.127197265625, 1.18359375, -1.9248046875, -0.630859375, 1.8046875, 0.88134765625, 0.9462890625, -0.5263671875, -0.27685546875, 0.50634765625, -0.09014892578125, -0.79443359375, -0.170166015625, 0.489501953125, 1.400390625, 0.51904296875, -0.10504150390625, 1.5, -1.099609375, 1.251953125, 0.1434326171875, 0.1802978515625, -0.4150390625, -0.517578125, -1.59765625, 0.378173828125, -0.408203125, -0.380615234375, 1.4970703125, 0.6640625, 0.316650390625, -0.76171875, -0.438232421875, 0.439208984375, -0.0006985664367675781, 1.10546875, -1.578125, 1.228515625, -0.1763916015625, -0.203369140625, -1.5830078125, 0.5966796875, 0.85498046875, -2.380859375, 0.7646484375, 1.109375, -3.44140625, 0.1298828125, -1.6494140625, 0.79541015625, -0.443603515625, -0.8291015625, -0.00395965576171875, 0.1630859375, 0.1798095703125, -1.6474609375, -1.5908203125, -1.169921875, -1.4345703125, 0.2335205078125, -0.91455078125, 1.314453125, 3.380859375, 0.2215576171875, -0.308349609375, 0.84423828125, 0.26708984375, 1.6494140625, 0.07025146484375, 0.81396484375, 1.408203125, 0.74267578125, -1.373046875, -1.431640625, -0.140380859375, -0.54638671875, 1.43359375, 1.44140625, 0.39111328125, -0.80810546875, -1.779296875, 0.314208984375, 2.921875, -0.453369140625, 2.833984375, -0.039703369140625, 0.205322265625, -0.1817626953125, -0.175048828125, 1.130859375, 0.60595703125, -0.344482421875, 0.075927734375, -0.025665283203125, 3.078125, -0.1138916015625, -0.65087890625, 0.145751953125, 0.9541015625, -0.7666015625, 0.70947265625, 0.29931640625, -0.54296875, 0.243408203125, -1.6591796875, 1.54296875, -0.90673828125, -0.3427734375, -0.1710205078125, -0.191162109375, -0.45458984375, 1.12109375, 1.2255859375, -1.4921875, -0.044586181640625, -0.916015625, 1.666015625, 1.294921875, 1.5576171875, 0.99072265625, 0.6123046875, -0.32666015625, 0.4326171875, -1.5966796875, -0.5341796875, -0.55419921875, 0.474853515625, -0.6123046875, -0.357421875, -1.201171875, 1.3447265625, -1.7236328125, 1.005859375, 1.0263671875, -0.0238494873046875, -0.51123046875, 1.1298828125, 0.29833984375, -1.7607421875, -0.95849609375, -1.0458984375, -1.5673828125, 1.0234375, -0.87890625, -0.974609375, -0.56298828125, -0.022857666015625, -0.055419921875, -0.55810546875, -1.708984375, 0.2607421875, 0.5615234375, 2.26953125, 0.759765625, -1.111328125, -0.50244140625, 1.5166015625, -1.20703125, -0.7119140625, -0.395751953125, 0.5498046875, 0.822265625, 0.923828125, 0.212646484375, -0.5751953125, -0.34423828125, 0.445556640625, -0.30908203125, 0.094482421875, 0.8916015625, 0.93701171875, 1.1943359375, 0.216064453125, 0.53369140625, -0.11114501953125, 0.27880859375, 0.04974365234375, 0.0259552001953125, -1.3974609375, 0.282470703125, -0.35205078125, -0.86572265625, 0.12548828125, -0.37841796875, -0.97119140625, -0.91748046875, 1.3095703125, 0.55810546875, 0.239990234375, 0.180419921875, 0.3291015625, 0.6162109375, 1.4453125, 0.280029296875, 0.285400390625, 1.5869140625, -1.0634765625, -0.2093505859375, -0.60595703125, 1.4599609375, -0.07513427734375, 0.271240234375, 0.152099609375, 0.11456298828125, -1.736328125, -0.64892578125, 0.8740234375, -1.2138671875, 1.4951171875, 0.257080078125, -0.94580078125, -1.9580078125, -0.91064453125, -0.97412109375, 0.921875, 0.0164794921875, -0.07623291015625, -1.2197265625, -0.693359375, -1.0146484375, 1.2109375, 1.314453125, 1.8349609375, 0.92529296875, 1.349609375, 1.205078125, 0.591796875, 0.9345703125, 0.3037109375, 0.1373291015625, 0.7802734375, 1.7099609375, -2.904296875, 0.10894775390625, 1.0517578125, -0.5732421875, -0.93701171875, 1.0244140625, -0.29248046875, -0.145751953125, -0.1563720703125, -0.86474609375, -0.433837890625, 0.65771484375, -0.55810546875, -0.79736328125, -0.8310546875, -0.34375, -2.416015625, 0.77392578125, 0.01425933837890625, 1.2685546875, 0.408203125, 0.429443359375, -1.357421875, -1.3662109375, -0.422607421875, 0.45751953125, -2.06640625, 0.044464111328125, 0.66064453125, -0.62158203125, 0.8056640625, -0.3505859375, 0.012115478515625, 0.53955078125, -0.66259765625, 0.681640625, -0.08526611328125, 0.654296875, -1.671875, -0.2181396484375, 0.86474609375, -0.0198211669921875, 0.86767578125, -1.0068359375, 1.0869140625, 0.69677734375, -0.41845703125, 0.37744140625, 0.7060546875, -0.2044677734375, 1.0576171875, -1.4765625, 0.869140625, 1.3388671875, 1.2158203125, 0.6572265625, 0.2783203125, -0.1767578125, 1.5859375, 0.1873779296875, 0.611328125, 0.4375, 2.044921875, 0.250244140625, 0.404296875, -0.876953125, 1.7158203125, -0.76611328125, 0.70654296875, -0.82861328125, -1.3466796875, 0.55322265625, 0.415771484375, 0.87548828125, 1.8037109375, 0.48193359375, 0.51513671875, -0.82470703125, 0.50146484375, 0.732421875, 0.84619140625, -0.51123046875, 0.26611328125, 1.1318359375, -0.7734375, -0.08074951171875, 1.0439453125, -0.179931640625, 0.83154296875, 0.55078125, 1.283203125, -1.779296875, -0.4375, -2.07421875, 0.345947265625, 1.369140625, -0.2032470703125, 0.77880859375, -0.66357421875, -1.271484375, -0.004383087158203125, -0.61572265625, -0.172607421875, -0.480712890625, -0.66943359375, -0.76904296875, -0.28662109375, 2.3984375, -1.3095703125, -1.9619140625, -1.1376953125, -0.95166015625, -0.2220458984375, 0.44677734375, 1.06640625, -0.1927490234375, 0.31005859375, -1.2685546875, -0.03076171875, -0.79443359375, 0.218017578125, -0.386474609375, -1.0126953125, 0.0009760856628417969, 0.79052734375, -2.291015625, -0.515625, 2.267578125, 0.364501953125, 0.36865234375, 1.7900390625, 1.2373046875, -0.076904296875, -0.340576171875, 0.18408203125, 0.82373046875, 1.7880859375, 0.254150390625, -0.417236328125, 0.8095703125, 0.78076171875, -0.35009765625, -1.21875, 0.689453125, 0.6630859375, -0.5771484375, 1.1728515625, 0.0173187255859375, -1.4921875, -0.279296875, -0.321044921875, 0.59326171875, -0.05780029296875, 1.548828125, 0.041717529296875, 0.388671875, 0.20166015625, -0.45556640625, 1.201171875, -0.77880859375, 1.8857421875, 0.57080078125, 1.41796875, 0.465576171875, -0.96923828125, 0.71533203125, -0.2379150390625, -0.1593017578125, 0.74072265625, 0.82421875, 0.37158203125, 1.0693359375, 0.91015625, 0.6943359375, -0.97998046875, 0.88671875, 0.7646484375, 0.7041015625, -0.6328125, -2.69140625, 0.0919189453125, 1.0361328125, -1.248046875, -2.208984375, -0.2122802734375, 1.52734375, -1.5625, -0.78125, -0.308837890625, -1.9912109375, 0.466796875, -0.51611328125, 1.021484375, -0.408203125, 1.4189453125, 2.16015625, -0.39453125, -0.607421875, -0.342529296875, -0.64306640625, -0.3232421875, 1.5185546875, 0.292724609375, 1.8388671875, 0.6611328125, 0.958984375, -1.2314453125, 0.1907958984375, 1.4267578125, -2.205078125, -1.21875, 0.82666015625, 0.2498779296875, -1.04296875, 0.0091705322265625, -0.3017578125, 0.92041015625, -0.7021484375, 0.24169921875, 0.325439453125, 0.3466796875, 0.00740814208984375, 0.71142578125, 0.35595703125, 0.199951171875, -1.486328125, 1.0419921875, -0.12396240234375, -0.412109375, 0.50634765625, -2.08984375, -0.75048828125, 0.52490234375, 1.0537109375, 0.37841796875, 1.75390625, 0.83203125, -0.568359375, -0.68505859375, 0.30517578125, 1.0283203125, -0.74365234375, 0.50830078125, 0.72119140625, 0.280029296875, 1.1416015625, -1.30078125, 0.51025390625, -1.0771484375, 0.63232421875, 0.428466796875, -0.50927734375, 2.61328125, 0.04827880859375, 1.326171875, 1.0615234375, -0.70068359375, 0.89892578125, 1.4267578125, 1.5478515625, 0.8779296875, -0.0672607421875, -0.299560546875, 0.46728515625, 0.68359375, -1.76953125, -0.12841796875, -0.10101318359375, -1.0439453125, 1.0078125, -0.82861328125, 0.48974609375, 1.0810546875, 0.2486572265625, 0.486083984375, 0.40087890625, -0.388916015625, -0.60498046875, 1.859375, -0.2685546875, 0.96337890625, -0.529296875, 1.07421875, 2.05859375, 0.321044921875, 0.544921875, 1.4560546875, 1.099609375, -1.361328125, -1.263671875, 0.607421875, -1.298828125, 1.833984375, -0.513671875, -0.05010986328125, 0.016632080078125, 0.08074951171875, -0.96630859375, 0.7744140625, 0.2156982421875, 0.6591796875, 0.52978515625, -1.4521484375, 0.5869140625, 0.1605224609375, -1.0322265625, 1.7822265625, 1.123046875, -0.279052734375, -0.7265625, 0.1297607421875, 0.7998046875, -0.04217529296875, 1.3984375, 0.185302734375, -0.5126953125, -0.01654052734375, 1.3359375, 0.032379150390625, -0.6923828125, -0.286865234375, 0.2021484375, 0.87939453125, 1.806640625, 1.1259765625, -1.923828125, 1.9716796875, -0.27978515625, -2.259765625, 1.6328125, -0.421630859375, 0.6943359375, -0.416259765625, -1.248046875, 0.411865234375, 0.318359375, 0.434326171875, 0.10516357421875, 0.3525390625, -0.044097900390625, 0.7705078125, -0.119140625, 1.05859375, -0.046966552734375, -0.431884765625, -0.55322265625, 0.44775390625, 0.32421875, 0.1737060546875, -1.45703125, -0.101318359375, 0.7138671875, -0.82958984375, -0.1607666015625, 0.34912109375, -0.0357666015625, 0.89794921875, 0.408447265625, 0.8427734375, 0.64892578125, -2.056640625, -0.7607421875, -1.3525390625, 0.5927734375, -0.275390625, 0.09051513671875, 0.04058837890625, -2.16015625, -0.1285400390625, 0.38427734375, -1.0810546875, -0.01922607421875, -0.370361328125, 0.359130859375, 0.62939453125, -1.9599609375, -2.640625, 0.2191162109375, -0.0201416015625, -1.3173828125, 0.88037109375, -0.483642578125, -0.31298828125, 1.1689453125, -0.26806640625, -0.06805419921875, -0.84033203125, -0.68896484375, 0.87255859375, 0.35400390625, -1.5771484375, 0.43408203125, 1.33203125, 0.95068359375, 0.4423828125, 2.16796875, -1.3310546875, 0.7392578125, 1.8916015625, 1.1162109375, -0.76416015625, 0.40966796875, 1.361328125, -2.275390625, -0.27392578125, -0.51513671875, -0.2049560546875, 0.0090789794921875, -0.143798828125, 0.74853515625, -0.73583984375, -0.007183074951171875, -0.39794921875, -0.2176513671875, -0.75732421875, 0.154541015625, 0.0202484130859375, -1.7177734375, -0.7607421875, 0.50390625, 1.1064453125, -0.11669921875, -0.724609375, -0.2064208984375, -0.07708740234375, 0.1590576171875, 0.3193359375, 1.7978515625, -0.67041015625, -0.88525390625, 1.01171875, 0.39697265625, 0.2152099609375, -0.1300048828125, -0.4111328125, -0.861328125, 0.48046875, -0.9384765625, -0.55810546875, -0.59375, -0.3779296875, -0.179443359375, 0.8984375, 0.0274658203125, 0.92431640625, -0.007106781005859375, -0.162109375, -0.58935546875, -2.447265625, -0.67431640625, 0.54248046875, 0.7197265625, 1.3486328125, -0.375, -0.82275390625, 1.044921875, -0.58984375, -1.0361328125, -1.1259765625, -0.27197265625, 1.197265625, 0.20947265625, -0.26220703125, -2.15234375, -0.2215576171875, -0.1533203125, -0.1492919921875, -0.207763671875, 2.19921875, -0.256591796875, 0.1298828125, -0.0121917724609375, 1.4013671875, 0.469970703125, 0.042388916015625, -0.65771484375, 0.4873046875, 1.5, 0.6494140625, -1.0234375, -0.241455078125, -0.2142333984375, 0.086181640625, 2.703125, 0.415283203125, 0.1114501953125, 0.422119140625, -0.0213775634765625, 0.80078125, -0.49951171875, -0.454345703125, -0.919921875, -0.381103515625, -0.09075927734375, 0.6875, -1.3232421875, 1.837890625, -0.1953125, -0.5810546875, 1.3798828125, -1.0546875, 1.4052734375, 0.6982421875, 0.986328125, 1.419921875, 0.7578125, -2.2265625, 0.36572265625, 0.1900634765625, -1.1904296875, 0.66748046875, 0.56494140625, -1.3095703125, 0.978515625, 1.447265625, -0.058074951171875, 0.77685546875, 1.02734375, -0.491455078125, -1.7900390625, -0.60791015625, 0.3203125, -1.1025390625, 0.27001953125, 1.2958984375, -0.1092529296875, -0.2069091796875, 0.38623046875, 0.43115234375, 0.33837890625, 1.1181640625, -1.1083984375, 0.293212890625, -2.17578125, -0.62646484375, -0.7763671875, 0.58740234375, 1.26171875, 0.2008056640625, 0.5703125, 0.18701171875, 0.943359375, -3.330078125, -1.5478515625, -0.947265625, -0.1488037109375, -0.2080078125, 0.86328125, -0.2802734375, 1.455078125, 0.9873046875, -1.658203125, 0.0195770263671875, 0.294677734375, 0.73583984375, 0.90771484375, -0.1790771484375, -1.322265625, -0.1673583984375, -1.564453125, -0.045135498046875, -0.145263671875, -0.861328125, -1.7958984375, 1.28125, 0.62255859375, 0.78857421875, -0.55126953125, 0.85693359375, -0.1097412109375, 1.3779296875, 0.08282470703125, -0.349609375, -1.6982421875, 0.31787109375, 0.37353515625, 1.7265625, -1.0361328125, -0.62890625, -0.059417724609375, -0.43212890625, -2.84765625, 1.015625, -0.352294921875, -1.6845703125, -1.22265625, 0.74267578125, -1.80859375, 1.0361328125, -0.7001953125, 0.68798828125, 0.11602783203125, -0.338623046875, 1.98828125, 1.0751953125, 0.81396484375, 1.076171875, 0.7841796875, 1.015625, -0.08551025390625, -1.11328125, 1.609375, 1.3662109375, -0.092041015625, 1.55859375, 1.759765625, -0.3623046875, 0.259033203125, -1.6591796875, -0.022491455078125, -0.7626953125, 0.12298583984375, 0.2493896484375, -0.76708984375, 0.93408203125, 2.439453125, -0.50244140625, 1.39453125, -0.3623046875, 1.41015625, 2.666015625, -0.0753173828125, 1.37890625, -1.0869140625, -0.312744140625, -1.9365234375, -0.3310546875, 1.3955078125, -0.93505859375, -0.35693359375, -0.5869140625, -0.87646484375, 0.01374053955078125, -0.038116455078125, -0.6904296875, 0.8154296875, -0.400146484375, 0.66455078125, 0.87890625, -0.77783203125, -1.1572265625, -2.16015625, -0.031036376953125, 0.541015625, -1.1181640625, -0.65966796875, -0.189208984375, -0.211181640625, -2.03125, -1.51953125, -0.028717041015625, 0.7744140625, 0.233154296875, -1.51953125, 0.677734375, 0.97412109375, -0.919921875, 0.38818359375, 0.041290283203125, 2.134765625, 0.962890625, -0.7978515625, -1.0830078125, -0.30029296875, 1.3203125, -1.294921875, -0.2900390625, -0.61767578125, 1.08203125, -0.48046875, -1.3564453125, -3.021484375, -1.4541015625, 1.0830078125, 0.343994140625, 2.33203125, 0.93505859375, -0.41455078125, 2.61328125, -0.113525390625, -0.39404296875, 0.053466796875, -0.865234375, 0.89990234375, -0.095703125, -1.3349609375, 1.5673828125, -0.88818359375, 0.37451171875, -1.150390625, -1.0673828125, -0.63037109375, 0.351318359375, -0.4052734375, 0.9423828125, 0.53369140625, -0.59814453125, 1.013671875, 0.462158203125, -0.044647216796875, -0.99951171875, -0.578125, 2.005859375, -0.98095703125, 1.1611328125, 0.755859375, -0.365478515625, -0.87158203125, -0.87646484375, 0.9619140625, -0.43212890625, -1.1591796875, 0.28076171875, 0.196533203125, 1.244140625, 0.448486328125, -1.0576171875, -0.19970703125, 0.5400390625, -0.865234375, -0.427490234375, 2.421875, -2.080078125, -0.07965087890625, 0.443603515625, -1.0576171875, 0.63427734375, -0.176025390625, -0.67138671875, -1.166015625, -1.7744140625, -0.1278076171875, 0.0635986328125, 0.0312042236328125, 2.587890625, -0.279052734375, 0.439697265625, 0.052825927734375, -0.7470703125, -0.0616455078125, -2.41015625, 1.2568359375, -0.5009765625, -0.11639404296875, -1.4111328125, -0.095947265625, -0.76123046875, 0.67626953125, 0.5224609375, -0.54443359375, 0.59619140625, 1.169921875, -0.1575927734375, 0.10528564453125, 0.5283203125, -0.80224609375, -1.1162109375, -0.05438232421875, 0.09552001953125, -0.06402587890625, 0.53125, -0.38134765625, -0.33984375, -1.3056640625, 0.196044921875, 0.452392578125, 0.11456298828125, 0.03070068359375, -0.5126953125, 1.8564453125, -1.3671875, -0.62109375, -1.115234375, 0.9462890625, -0.5009765625, -1.119140625, 0.346435546875, 0.31396484375, -1.2392578125, 1.7041015625, 1.078125, 0.2073974609375, 0.76025390625, 0.79443359375, -0.235107421875, 1.515625, -0.440673828125, 0.73095703125, 1.8544921875, 0.382568359375, 0.349853515625, 1.423828125, -0.86376953125, 1.6201171875, 0.59130859375, 0.81591796875, 0.86572265625, -2.5078125, 0.982421875, -2.4453125, -0.55322265625, 0.40234375, -1.294921875, 0.0114288330078125, 0.52294921875, -0.306640625, 1.6328125, -1.189453125, -0.445556640625, -0.076171875, 0.9501953125, 0.748046875, 0.78662109375, 0.53515625, 0.454345703125, -0.0869140625, 0.1319580078125, -1.6162109375, -0.81005859375, -1.533203125, -0.08447265625, -1.97265625, -0.250244140625, -0.546875, 0.27099609375, 0.238037109375, -0.71630859375, -0.63134765625, 1.0498046875, -0.3369140625, 0.921875, 0.86328125, 1.7412109375, -0.47900390625, -1.3984375, -0.1697998046875, -0.1263427734375, 1.5732421875, 0.58935546875, -0.5576171875, 0.51123046875, -0.345458984375, 0.00311279296875, -0.397216796875, -0.88427734375, 0.892578125, -0.974609375, 0.478271484375, -0.97021484375, 0.748046875, -0.30078125, 1.6044921875, -0.68505859375, -0.089111328125, -1.4228515625, -1.9912109375, 0.865234375, 0.083984375, -0.305419921875, -0.5283203125, -0.395263671875, -0.830078125, -0.493408203125, -0.315673828125, 1.0556640625, -0.270751953125, -0.468505859375, 0.9521484375, 0.5576171875, 0.5322265625, 1.1669921875, -0.25634765625, -0.2235107421875, 0.2291259765625, 0.291748046875, 1.0595703125, 0.378173828125, -0.1573486328125, 1.7900390625, 1.9814453125, 0.990234375, -0.352294921875, 1.0615234375, 0.28515625, 1.0009765625, 1.6787109375, 0.59326171875, -0.19482421875, -0.30615234375, 0.8896484375, 0.79833984375, -0.66455078125, 1.388671875, -1.01171875, -0.703125, 1.0537109375, 0.97607421875, -1.2548828125, -0.7607421875, -0.560546875, -0.78271484375, -0.47998046875, -0.83154296875, 0.1402587890625, 1.013671875, 0.1878662109375, 0.2005615234375, 0.583984375, 1.4052734375, -0.4990234375, -1.5, -1.0947265625, -0.061126708984375, 0.377685546875, 0.393310546875, -0.38134765625, 1.7490234375, 1.8232421875, -0.466552734375, 1.23046875, -0.01140594482421875, 0.06414794921875, 0.79833984375, 0.275146484375, -1.5634765625, 0.4375, -2.28125, 0.332275390625, -1.474609375, -0.2060546875, -0.82958984375, -0.2161865234375, -0.429931640625, 0.373291015625, 0.294189453125, 0.3212890625, -1.2236328125, -1.265625, 0.414794921875, -0.223388671875, -0.31982421875, 1.7958984375, 0.1488037109375, 0.50537109375, -0.3427734375, -0.375732421875, 0.4296875, -0.028289794921875, -0.060302734375, -0.235107421875, 0.280029296875, 2.447265625, -0.042877197265625, -1.0546875, 0.149658203125, -0.1070556640625, -1.3408203125, 0.357421875, 0.294189453125, 0.50390625, 0.5234375, 1.7626953125, -0.72265625, -0.281982421875, -0.4306640625, 0.77587890625, -0.57177734375, -1.1318359375, -1.138671875, -0.9755859375, -0.478271484375, -0.9462890625, -0.64208984375, 0.970703125, -0.61474609375, -0.323974609375, -0.07232666015625, 0.9501953125, 1.2314453125, 0.261474609375, 0.109130859375, 0.211181640625, 1.1455078125, -0.2177734375, 0.7490234375, 0.193359375, 0.290283203125, 0.8662109375, 0.541015625, 0.11260986328125, 1.603515625, -0.9130859375, 0.78662109375, -0.57861328125, -0.80126953125, -1.6982421875, -0.449951171875, -0.72216796875, -0.44775390625, 0.71533203125, 0.7314453125, -0.335205078125, 0.361083984375, 0.483642578125, 1.5107421875, -1.1611328125, 0.716796875, -0.09552001953125, -0.157470703125, -0.180419921875, 1.404296875, -0.5498046875, -0.129150390625, 1.427734375, -0.64794921875, -0.80322265625, 0.11212158203125, 0.93994140625, -0.340087890625, -1.5966796875, -0.370849609375, -1.3974609375, -0.46044921875, -1.2275390625, -2.654296875, -2.03515625, -0.319091796875, -0.396728515625, -0.56787109375, 1.607421875, -0.0195770263671875, -0.1712646484375, 0.58154296875, 0.11614990234375, 0.89599609375, -0.00047588348388671875, 0.5126953125, 1.6220703125, 1.2421875, -0.452392578125, -1.0947265625, 0.578125, -0.84375, -0.57080078125, -1.291015625, -0.248779296875, 0.79443359375, -0.109619140625, -0.1019287109375, 0.79638671875, -1.1875, 1.6982421875, 0.55810546875, -1.306640625, 0.65576171875, -0.143310546875, 0.486328125, 0.30029296875, 0.86279296875, 0.9091796875, -0.7294921875, 0.323974609375, 1.044921875, 2.0234375, 0.216064453125, -0.333251953125, 1.44921875, 0.35107421875, 0.734375, 0.54345703125, -0.541015625, 0.2161865234375, 0.501953125, -0.38623046875, 0.3583984375, 1.1435546875, 0.1177978515625, -1.2294921875, -0.51123046875, 1.6240234375, 0.73974609375, 0.240478515625, 1.9541015625, -0.475341796875, -0.51318359375, 0.4833984375, -0.33203125, -0.32373046875, -0.62646484375, -0.32421875, -0.55615234375, -0.86328125, -1.3916015625, 0.291748046875, -1.2646484375, -0.96728515625, -0.35107421875, -0.83251953125, 0.50341796875, 0.06439208984375, 0.9404296875, -1.8818359375, 0.273193359375, 0.6728515625, 0.80078125, -1.318359375, 0.87646484375, -1.8974609375, 0.71533203125, 0.39697265625, 0.6669921875, -0.654296875, -0.08929443359375, -2.2265625, 0.470458984375, 1.2607421875, 2.228515625, -0.7412109375, 0.18603515625, 0.4990234375, 0.1260986328125, -1.9287109375, -0.82666015625, 1.5673828125, 0.1536865234375, 1.53125, -0.60009765625, 0.0316162109375, 0.473876953125, 0.04266357421875, -0.482666015625, -1.255859375, -0.12408447265625, 0.2734375, 1.2294921875, -0.370361328125, 0.708984375, -0.427734375, -0.91357421875, -1.025390625, 0.84912109375, 0.476318359375, -0.452880859375, 1.17578125, -0.44873046875, -0.49462890625, 0.206787109375, 0.2005615234375, 0.775390625, 1.3173828125, -1.12109375, -1.080078125, 1.0146484375, -0.365234375, -2.41796875, 2.0625, -2.916015625, 1.123046875, 0.9443359375, 1.0732421875, -0.30859375, -0.49072265625, -1.27734375, 0.379150390625, 1.2021484375, -0.01003265380859375, -0.37158203125, 0.0484619140625, 0.462646484375, -0.8837890625, 0.46875, -0.231201171875, 0.79150390625, 1.046875, 0.34326171875, -0.845703125, -0.517578125, -0.85400390625, -0.048126220703125, -0.87646484375, -0.13037109375, -0.08367919921875, -0.1282958984375, 0.040252685546875, -0.095947265625, -0.0160064697265625, -1.4794921875, -1.4052734375, -1.365234375, -1.5068359375, -0.3369140625, 0.59912109375, 0.1558837890625, -1.189453125, -0.42529296875, -0.94873046875, 1.431640625, -1.2666015625, 0.7548828125, -1.970703125, 0.261962890625, 0.09613037109375, 0.37353515625, 0.1781005859375, 0.6767578125, 1.8310546875, 0.4267578125, 1.025390625, -0.1551513671875, -1.501953125, 0.78955078125, 0.04718017578125, 1.2724609375, 1.2080078125, -2.376953125, -1.4453125, 0.006317138671875, 0.409423828125, 1.0400390625, -1.4853515625, -1.318359375, -0.1099853515625, 1.416015625, 0.191650390625, -0.90771484375, 1.43359375, 1.0654296875, 0.154052734375, -1.138671875, -0.82177734375, 1.625, 0.7919921875, -0.93017578125, 1.1875, 1.0859375, -0.68994140625, 0.78369140625, -0.045654296875, -0.302001953125, 1.0361328125, 1.24609375, 0.1839599609375, 0.890625, 1.310546875, 1.291015625, -1.498046875, -0.6171875, -0.9033203125, -0.32666015625, -0.66015625, -0.794921875, 1.009765625, -2.58984375, -0.261474609375, 0.054901123046875, 0.39794921875, 1.568359375, -0.06427001953125, 1.0673828125, 0.088134765625, 1.451171875, 0.489501953125, 0.2327880859375, 0.25634765625, 0.2081298828125, -0.11004638671875, -1.0673828125, 0.4638671875, -1.4208984375, 1.720703125, -0.1632080078125, 1.306640625, 1.0869140625, -0.51220703125, 0.11505126953125, 0.81494140625, -0.0733642578125, -1.5322265625, -0.196044921875, 0.281005859375, -1.541015625, 0.8564453125, 0.30126953125, 0.09918212890625, -1.1533203125, 0.453857421875, 1.0654296875, 1.84765625, 0.525390625, 0.6572265625, 0.27490234375, -0.79345703125, 1.2451171875, 0.689453125, -0.1787109375, -0.47998046875, -1.0712890625, 3.029296875, 1.1005859375, 0.1507568359375, 0.8076171875, 0.224365234375, 0.96142578125, -1.673828125, 0.05108642578125, 1.359375, 0.0968017578125, 0.95068359375, -0.06903076171875, -1.3232421875, -0.215576171875, 1.47265625, 0.2763671875, 1.7216796875, 0.59765625, 0.724609375, -0.7236328125, -0.2437744140625, 0.33642578125, 1.3955078125, 1.1328125, -1.26953125, -0.1756591796875, -0.52978515625, -1.560546875, -0.73388671875, 0.2705078125, 1.1533203125, 1.0283203125, -0.32177734375, 1.736328125, 0.09063720703125, 1.03125, 0.99072265625, -0.32275390625, 0.5634765625, -2.5234375, 0.2303466796875, -1.193359375, 1.759765625, -0.52392578125, 0.1319580078125, -0.148681640625, 0.80712890625, 0.7548828125, 0.68310546875, 0.7490234375, -0.209716796875, 0.347900390625, 1.4970703125, -1.0859375, -1.3818359375, 1.279296875, 1.1357421875, 0.4453125, 1.7353515625, 1.7080078125, -0.92431640625, -0.0584716796875, 0.25732421875, -0.2005615234375, 1.0166015625, -0.94580078125, 0.37890625, -0.76513671875, 1.2548828125, -0.41748046875, 0.9130859375, 0.10015869140625, 0.3330078125, 1.0341796875, -1.3046875, 0.7705078125, -0.59228515625, -0.529296875, -1.0390625, 0.213134765625, -0.4228515625, -0.98681640625, 0.00110626220703125, 0.359130859375, 0.1639404296875, -0.1636962890625, 0.0035953521728515625, 1.5185546875, -0.69091796875, -0.59619140625, 0.671875, -0.53857421875, -0.01959228515625, -0.75537109375, -1.4228515625, 0.95751953125, -0.74658203125, -0.08966064453125, -1.1142578125, -0.66748046875, -1.9140625, 0.6162109375, -0.501953125, 1.15234375, -1.0830078125, 1.115234375, -1.244140625, 1.4931640625, -1.681640625, 0.9912109375, 0.39697265625, 0.7119140625, -0.95068359375, 0.456298828125, 0.30224609375, 0.57958984375, -0.041412353515625, 0.1591796875, 0.50830078125, 0.68994140625, 0.2257080078125, -1.677734375, -1.7568359375, 1.20703125, -1.0068359375, -1.0654296875, 0.0172882080078125, 0.49267578125, 0.0921630859375, 1.1064453125, -1.6611328125, -0.56396484375, -0.123291015625, -0.85888671875, 0.0718994140625, -1.11328125, -1.5390625, 0.58837890625, 1.525390625, -1.0615234375, 0.5771484375, -0.334716796875, -1.3076171875, 1.3037109375, 0.40771484375, 0.00835418701171875, 0.393798828125, -1.6640625, -2.09375, -0.0321044921875, -0.266845703125, -0.1534423828125, 0.85693359375, -0.775390625, -0.3359375, -0.364013671875, -0.50830078125, -1.1005859375, 0.8359375, 0.75244140625, 1.0595703125, -0.33837890625, -0.2529296875, 0.4775390625, -1.033203125, -1.7353515625, -1.5615234375, 0.53759765625, 0.382080078125, -0.06640625, 0.06341552734375, -0.447021484375, 1.1015625, 0.302490234375, -2.5546875, -0.74658203125, -1.5888671875, 0.0546875, 0.572265625, -1.5849609375, 0.481689453125, 0.452392578125, 1.681640625, 2.025390625, 0.306396484375, 0.270751953125, 0.7734375, -0.45458984375, -0.7861328125, 1.0009765625, -0.438232421875, 1.1533203125, 0.0635986328125, 0.8310546875, -2.677734375, 1.9755859375, -1.046875, -0.5478515625, 0.448974609375, -0.31103515625, 1.3837890625, -1.1611328125, 0.30517578125, 0.8681640625, -0.015869140625, -1.880859375, -1.130859375, -1.869140625, 1.318359375, 0.336669921875, 0.97607421875, -0.001399993896484375, 1.103515625, 1.466796875, -0.87548828125, -0.2232666015625, -0.34375, 1.9658203125, -0.85791015625, 0.391845703125, 0.63037109375, -0.11309814453125, -0.080078125, -0.7109375, -0.020416259765625, 0.5791015625, 0.33154296875, -1.322265625, 0.55810546875, -1.251953125, 0.12481689453125, -0.91455078125, -1.044921875, 0.50244140625, -0.7158203125, 0.425537109375, -0.9873046875, -1.6337890625, 2.123046875, 0.3701171875, 1.298828125, -0.2076416015625, 0.53125, -0.2447509765625, -0.5263671875, -0.180908203125, 0.85205078125, -0.31298828125, -2.171875, 0.634765625, 1.5576171875, -0.0948486328125, 0.30810546875, -1.1328125, -0.321533203125, -0.1490478515625, -0.71630859375, 0.85400390625, -0.12255859375, 0.04376220703125, -0.071533203125, 0.25244140625, 0.85107421875, 0.234130859375, -0.02679443359375, -1.998046875, -0.306884765625, 1.5703125, -0.53125, 0.71923828125, 1.4677734375, -1.005859375, 0.070068359375, -0.413330078125, 0.019683837890625, -0.434814453125, -0.2275390625, -0.55712890625, -1.1171875, 0.71826171875, -1.4453125, 1.6181640625, -0.4384765625, 1.5673828125, -0.0830078125, 1.310546875, -0.10595703125, -0.4873046875, -1.123046875, 1.3310546875, 0.275390625, 0.04150390625, 0.122802734375, -0.043365478515625, 0.04766845703125, 1.7109375, 0.9013671875, -0.331787109375, 1.6953125, 1.1396484375, -0.22705078125, -1.16796875, -0.445556640625, 1.123046875, 1.978515625, -0.83740234375, -0.88427734375, -0.324951171875, -0.76171875, -1.2880859375, -0.037750244140625, 0.431640625, -0.55908203125, -0.0295562744140625, -1.736328125, -0.52685546875, -0.78955078125, -0.2841796875, 0.446044921875, -0.3662109375, 1.01171875, -1.08203125, -0.939453125, 1.162109375, -1.1767578125, -0.4765625, 0.82861328125, -0.92578125, 1.94921875, -0.09228515625, -0.307373046875, 0.298583984375, 0.1590576171875, -0.288330078125, -0.255615234375, -0.80859375, 1.515625, 1.8173828125, -0.62890625, -1.833984375, -0.6689453125, 0.91796875, 0.76025390625, 0.71142578125, -1.3310546875, 0.245361328125, 0.2322998046875, 1.890625, 0.6767578125, 0.091552734375, -1.28125, 1.8603515625, 0.1297607421875, 0.56640625, -0.2041015625, -0.10791015625, -0.393798828125, 2.4609375, 1.015625, -0.2841796875, 1.2138671875, 0.09796142578125, -0.441650390625, 0.1168212890625, -0.67822265625, 0.8115234375, -0.1064453125, -1.55859375, 1.140625, -0.492919921875, -0.250732421875, 0.53173828125, 0.38720703125, -1.6005859375, -1.224609375, -0.6884765625, -0.468017578125, -0.153564453125, -0.32177734375, 0.423828125, 1.1357421875, -1.4404296875, -0.41748046875, -0.257568359375, 0.11273193359375, -2.017578125, 0.5751953125, 0.61767578125, 0.81884765625, 1.033203125, 0.10833740234375, 0.46240234375, -0.1368408203125, 0.7080078125, -0.69677734375, -1.0, 0.393798828125, -0.322509765625, 0.245361328125, 0.1990966796875, 1.26171875, -0.286376953125, -1.5888671875, 0.4248046875, 1.658203125, -1.658203125, 1.7646484375, -0.6650390625, 1.09375, -0.72021484375, -2.365234375, 1.033203125, 1.9453125, -1.4814453125, -0.97412109375, -0.383544921875, 0.97265625, -0.422607421875, -1.7685546875, 0.6015625, 0.183349609375, -0.208740234375, -0.379150390625, -0.88232421875, 0.407470703125, 1.6123046875, 0.27587890625, 1.2998046875, -0.81103515625, 0.00463104248046875, 0.568359375, 1.03125, 0.6962890625, -0.4990234375, 0.337158203125, 0.0032405853271484375, -0.09112548828125, -0.93115234375, 0.486083984375, 0.57470703125, 0.249267578125, 1.349609375, 0.74853515625, -0.31005859375, -0.400146484375, 1.662109375, 0.1815185546875, 2.490234375, 0.1683349609375, 0.55908203125, -2.0234375, -0.44140625, 0.158203125, 0.0266265869140625, 0.830078125, -1.484375, 1.16015625, 0.7119140625, 0.341552734375, 0.72998046875, 1.611328125, -1.15625, 0.2274169921875, -1.8779296875, -0.2071533203125, 0.3115234375, -0.189208984375, -1.1484375, 0.25244140625, -0.0214691162109375, 0.04962158203125, -1.61328125, 1.9052734375, 1.1748046875, -0.84619140625, -1.0546875, -0.775390625, 0.50390625, -1.9931640625, 0.342041015625, 0.73583984375, 0.51220703125, -0.39794921875, 1.3125, -0.556640625, -1.1572265625, 1.0361328125, 0.367919921875, -1.408203125, -1.4853515625, -0.084228515625, -1.16796875, -0.1221923828125, 0.93603515625, -0.3291015625, -0.6123046875, 0.53125, -0.2408447265625, -0.9140625, 0.57958984375, -1.2734375, -0.1871337890625, 1.1669921875, 0.295654296875, -1.2431640625, -1.44140625, -0.031463623046875, -1.33984375, 2.224609375, -0.2073974609375, -0.74267578125, -0.0233154296875, 0.74560546875, 0.92529296875, -0.62548828125, 0.93310546875, -0.70263671875, 1.8447265625, 0.80859375, 0.5576171875, -0.67431640625, 0.5693359375, -1.1513671875, 1.646484375, -0.443603515625, 0.716796875, 0.59228515625, -0.8310546875, -0.61279296875, -0.412353515625, -0.748046875, 0.009490966796875, -0.045928955078125, -1.6826171875, 0.6904296875, 0.09954833984375, -0.1495361328125, -1.4853515625, 1.052734375, 0.5078125, -0.479248046875, 1.5810546875, 1.0517578125, 1.1259765625, -0.6357421875, -0.147216796875, -1.7783203125, 2.068359375, 0.25146484375, -1.064453125, 0.912109375, -0.434814453125, -0.26171875, -0.4765625, 0.9775390625, 0.89111328125, 0.89501953125, -0.42138671875, -0.73876953125, 0.0210113525390625, -0.8837890625, 1.0224609375, 1.1962890625, 0.2052001953125, -0.4921875, -0.1580810546875, 1.23046875, 0.287353515625, 0.630859375, -1.4716796875, 0.7802734375, -0.48193359375, -0.29541015625, -0.1414794921875, -0.5439453125, 2.14453125, 0.58203125, 0.316650390625, -0.5224609375, 1.9169921875, -1.5029296875, 0.2177734375, -0.2158203125, -0.09710693359375, -1.859375, -0.60302734375, -0.4482421875, -0.95458984375, 0.1343994140625, 0.468017578125, 0.309814453125, 0.294921875, -0.32666015625, 1.1708984375, 0.1419677734375, -0.367919921875, 1.384765625, -0.4658203125, 0.290771484375, 0.705078125, 0.1453857421875, 0.48388671875, 1.3369140625, -1.4111328125, -1.083984375, 0.66552734375, 1.5244140625, 0.791015625, 0.8095703125, -0.453857421875, 0.2003173828125, 0.810546875, 1.6337890625, 0.70849609375, 0.227294921875, 0.25732421875, 2.0703125, 0.16650390625, 0.8232421875, 1.3076171875, 1.3369140625, -0.7666015625, 0.286865234375, -0.57080078125, -0.323974609375, -2.248046875, 0.99609375, -0.59375, -0.41162109375, 0.2156982421875, -0.56494140625, -0.10321044921875, 0.176025390625, 0.8251953125, -0.59228515625, -0.261474609375, -1.4189453125, 2.099609375, 0.400390625, -0.416015625, 1.5146484375, -1.69140625, 1.044921875, 1.0341796875, -0.30029296875, 1.134765625, -0.207763671875, -0.2015380859375, -0.732421875, 0.69970703125, -0.1307373046875, 1.3701171875, -0.8251953125, 1.7646484375, -0.468017578125, -2.201171875, 0.72705078125, -0.7255859375, 1.828125, 1.5234375, 0.173828125, 0.1890869140625, 1.296875, -0.2406005859375, 0.389404296875, 0.05279541015625, 0.275634765625, 0.2958984375, -0.14111328125, 1.328125, -2.2734375, -0.90185546875, 0.7880859375, -1.7470703125, -2.517578125, -0.78515625, -0.285888671875, -0.420654296875, -0.466064453125, -1.1962890625, 0.76953125, 1.048828125, -0.000732421875, -0.0267181396484375, -1.865234375, -0.6552734375, -1.0, -1.462890625, -0.55859375, -1.103515625, -2.447265625, -1.0810546875, -0.29931640625, -0.8369140625, -0.7939453125, -1.7998046875, -1.689453125, 0.33544921875, 0.1566162109375, 0.359130859375, 2.869140625, -1.41796875, -0.302490234375, -0.2841796875, -0.01264190673828125, 1.494140625, 0.84619140625, -1.669921875, 2.630859375, 1.69921875, 0.27880859375, -0.260009765625, -0.80517578125, -1.3505859375, 0.12152099609375, 0.052215576171875, 0.87841796875, -0.74658203125, -0.755859375, 1.3076171875, 0.68115234375, 0.29833984375, -0.7890625, 2.384765625, -0.355224609375, -0.44921875, -1.8076171875, 1.39453125, 2.943359375, 0.7451171875, -0.857421875, -0.52001953125, -0.04461669921875, -0.8984375, 0.59521484375, 0.043853759765625, 2.490234375, -1.4013671875, 0.55712890625, 0.826171875, 1.052734375, -1.255859375, 0.955078125, -1.2197265625, 0.388671875, -0.91357421875, 2.021484375, 1.3408203125, 0.0218658447265625, 0.40185546875, 2.7109375, -0.94775390625, -0.78466796875, 0.1375732421875, -0.50048828125, 0.72705078125, -1.6484375, -0.59375, 1.01953125, 1.126953125, 0.8154296875, -1.798828125, -0.5361328125, -1.0078125, 1.63671875, -0.285400390625, 2.11328125, -0.1490478515625, -1.072265625, 0.291259765625, 0.27734375, 1.0185546875, 0.337158203125, 0.12060546875, 0.1480712890625, -1.5390625, 0.400390625, -1.8486328125, -0.75048828125, 0.50830078125, 1.39453125, -0.9189453125, -0.10516357421875, 0.1441650390625, 0.7236328125, 0.53564453125, 1.24609375, 0.473876953125, -0.1845703125, -0.6064453125, -0.322509765625, -0.412353515625, -1.029296875, 0.216064453125, 0.9033203125, -0.49267578125, -2.1640625, -1.828125, 0.3427734375, 1.01171875, -0.451904296875, -1.3720703125, -0.92529296875, -0.75390625, 2.423828125, 0.9541015625, -0.91015625, -0.1943359375, 1.6357421875, -1.384765625, -0.69287109375, 0.54833984375, -0.408203125, -0.60107421875, 0.86962890625, 0.27294921875, -0.2529296875, 0.2030029296875, 1.916015625, 0.50634765625, 0.322998046875, 0.68359375, 0.8193359375, 0.662109375, -1.1884765625, 0.1365966796875, 0.04571533203125, -1.759765625, 0.91943359375, 0.04345703125, 0.066650390625, -0.00524139404296875, 1.6376953125, -1.0791015625, -0.74267578125, 0.654296875, 0.144775390625, -1.1005859375, 1.3056640625, 0.603515625, 0.630859375, -1.5634765625, -0.378173828125, -0.04547119140625, -2.056640625, -0.966796875, 1.03125, -0.353271484375, -0.2030029296875, -1.400390625, -0.439697265625, 0.354248046875, 0.46923828125, -2.6640625, -0.96240234375, 0.96533203125, -1.0078125, 0.443359375, 1.001953125, -0.71728515625, 1.31640625, -0.6904296875, 0.55517578125, -0.497802734375, -0.78662109375, -0.31005859375, 1.04296875, 0.413818359375, -0.4541015625, -0.06793212890625, -0.4658203125, 1.40625, -0.609375, 1.197265625, 0.9580078125, 2.125, 0.07025146484375, -1.20703125, -2.56640625, -0.568359375, 0.485107421875, 0.1602783203125, -0.57958984375, -0.0670166015625, 0.463623046875, -0.5712890625, -1.150390625, -1.0732421875, 0.004985809326171875, 2.005859375, -0.0300445556640625, 0.98486328125, -1.5654296875, 1.9072265625, -0.2159423828125, 0.48486328125, -0.19140625, -1.439453125, 0.1927490234375, -0.79931640625, -0.409912109375, 0.75048828125, 0.966796875, 0.5048828125, -0.240478515625, 0.2178955078125, 0.322998046875, -1.478515625, -2.353515625, -0.9423828125, 0.67333984375, 0.90478515625, -1.3984375, 0.0567626953125, 0.09783935546875, 0.0203857421875, -0.09600830078125, 0.30224609375, -0.490966796875, -0.39306640625, -2.09375, -0.85205078125, 1.0732421875, -0.04669189453125, -0.296630859375, 0.11865234375, -0.497314453125, 0.7333984375, -0.07830810546875, -1.978515625, -0.395751953125, -0.1416015625, 0.2325439453125, 1.7568359375, 0.43505859375, -0.64306640625, -0.830078125, -0.383544921875, 0.382080078125, -0.33837890625, -0.1163330078125, 2.02734375, 1.0146484375, -0.431884765625, -1.94140625, -0.58056640625, -0.798828125, -0.06964111328125, 2.23046875, 1.7158203125, 0.67041015625, 0.7744140625, 0.177001953125, -1.1181640625, -0.467041015625, 0.88818359375, -1.3671875, -1.330078125, 1.513671875, 0.625, -0.30126953125, -0.8408203125, -0.1463623046875, -1.4677734375, 0.00493621826171875, -1.4619140625, 0.8388671875, -0.71484375, -0.52392578125, 0.485107421875, -0.46533203125, 3.47265625, -0.407958984375, -0.1336669921875, 0.2178955078125, 0.3115234375, -1.6767578125, 0.1944580078125, 1.900390625, -0.311279296875, 0.388916015625, -0.4384765625, -1.3974609375, -0.360107421875, 0.07342529296875, -0.67333984375, -1.6298828125, -0.82275390625, -0.50830078125, 0.65087890625, 0.56982421875, 0.2354736328125, -0.56494140625, -0.129150390625, -0.0987548828125, -1.072265625, 1.171875, 0.367919921875, -1.4541015625, 0.264892578125, 1.0810546875, 1.7587890625, -0.425048828125, 0.322998046875, -1.19140625, -0.277099609375, 0.49072265625, -0.313232421875, -0.70458984375, -0.3701171875, 0.5400390625, -1.7998046875, -0.0911865234375, -1.1943359375, 0.294677734375, 0.8388671875, 0.86767578125, -0.32080078125, 0.1668701171875, 0.55029296875, -0.45654296875, 0.64794921875, -1.240234375, 0.560546875, 0.361572265625, 1.10546875, -1.296875, -0.300048828125, -1.076171875, -0.18212890625, 0.814453125, -0.10888671875, -0.043121337890625, -0.9345703125, 0.1793212890625, 1.5205078125, 1.408203125, 0.67919921875, 0.1861572265625, 0.146728515625, -1.69140625, 0.61376953125, 1.2724609375, -0.9013671875, 0.427978515625, 0.88134765625, -0.99462890625, 0.74853515625, 0.34423828125, -1.5830078125, 0.9326171875, 0.371826171875, 0.3720703125, 1.283203125, -0.7646484375, -0.0252685546875, 0.6064453125, -1.4580078125, 0.599609375, 1.3095703125, 1.296875, 0.580078125, 0.74853515625, -0.99951171875, 0.6171875, 0.259033203125, -0.8583984375, 1.126953125, -0.4208984375, -0.54638671875, 1.296875, 1.0732421875, 0.427978515625, 0.342529296875, 0.8154296875, -1.255859375, 0.03436279296875, 1.03515625, 0.3330078125, 0.0667724609375, 0.23095703125, -1.03515625, -1.5732421875, -0.99951171875, 1.236328125, -1.0244140625, 1.83203125, -0.3837890625, -0.64697265625, -1.4736328125, -1.353515625, -0.791015625, -0.76806640625, 0.52099609375, -2.6953125, -0.041748046875, -0.005687713623046875, -0.22021484375, 0.246337890625, 0.2802734375, -1.6181640625, 1.076171875, -0.416748046875, -1.5634765625, 0.64453125, -1.2353515625, 1.0869140625, -0.08416748046875, 0.5869140625, -1.431640625, 1.1630859375, -0.0179901123046875, -1.345703125, -0.033935546875, -0.83935546875, -0.1759033203125, 1.60546875, -1.28515625, -0.73876953125, 1.298828125, -0.89501953125, -0.413330078125, -0.419677734375, -0.042633056640625, 0.66552734375, 1.279296875, 0.3837890625, 0.48828125, -0.052642822265625, 0.85302734375, 0.7939453125, 1.1572265625, -0.168212890625, 0.4599609375, -0.75146484375, 2.72265625, 0.092041015625, 0.63525390625, -0.818359375, 0.837890625, 1.025390625, -0.83447265625, 0.29052734375, 0.853515625, 1.0634765625, 0.76171875, 0.336181640625, 0.1416015625, 0.94775390625, 0.1220703125, -2.587890625, -0.06988525390625, 2.12109375, 0.8388671875, -1.380859375, -0.70849609375, -0.162841796875, 0.7197265625, -2.005859375, -0.41650390625, 0.5693359375, -0.58642578125, 0.129638671875, -0.455322265625, -0.249755859375, -0.96923828125, 2.35546875, -0.499755859375, 0.26171875, 0.8291015625, 0.0150146484375, -0.65380859375, -0.4658203125, 0.3408203125, 0.62939453125, -1.34765625, 0.97802734375, 0.841796875, 1.5908203125, 2.169921875, 1.1923828125, -1.34375, 1.8369140625, -0.93994140625, 0.8369140625, 1.599609375, -0.68115234375, -1.1376953125, 0.6865234375, -1.5400390625, 0.99853515625, -0.2489013671875, 0.56396484375, 0.51171875, 1.24609375, -0.28857421875, 0.9462890625, 0.60595703125, 0.9658203125, 1.111328125, -0.9541015625, 0.5654296875, 1.3984375, 0.45361328125, -0.283935546875, -0.64794921875, -1.4423828125, -1.283203125, 0.33056640625, -0.399658203125, -0.541015625, -0.225830078125, -0.254150390625, -1.0703125, 0.8095703125, 0.41162109375, 0.0816650390625, 0.77392578125, 0.685546875, -0.3037109375, -0.6064453125, -0.362060546875, 0.03985595703125, -0.486083984375, 0.6123046875, 0.49609375, -1.529296875, -1.7919921875, -0.216064453125, 0.1627197265625, -0.32666015625, 0.083251953125, 0.12396240234375, -1.705078125, 0.42822265625, -1.8994140625, -1.0673828125, 1.951171875, 0.480224609375, -0.6572265625, 0.8154296875, 0.67529296875, -0.89794921875, 0.21728515625, 0.69384765625, 1.18359375, 0.69921875, 1.3564453125, -0.73291015625, -0.499755859375, 0.57763671875, -0.85546875, -2.669921875, -0.4794921875, -0.09271240234375, 1.6181640625, -1.095703125, -1.5029296875, 0.73388671875, 1.1513671875, 1.189453125, -0.93115234375, -0.7265625, 1.8408203125, -1.1220703125, 0.58984375, -0.95068359375, 0.0826416015625, -1.78125, -0.2293701171875, -0.81640625, -0.328369140625, 0.94482421875, -0.3203125, 1.3291015625, -0.0643310546875, 1.56640625, -1.33203125, -0.5537109375, -0.837890625, -1.0380859375, 0.27490234375, 0.424560546875, 0.085205078125, -0.59716796875, 0.447998046875, -0.3515625, 0.300048828125, -1.869140625, -0.34375, -2.05078125, -0.59912109375, 1.73046875, 2.33984375, -1.505859375, -0.83447265625, 0.6171875, -0.69140625, -0.74951171875, -1.611328125, -0.270751953125, -0.3056640625, -0.83935546875, 2.232421875, 0.1617431640625, 0.64306640625, 0.92431640625, 0.931640625, 0.31201171875, -0.56494140625, 0.6787109375, -0.41455078125, 0.33447265625, 0.397705078125, 0.42041015625, -1.53125, 1.859375, -0.79931640625, 1.5791015625, 0.1605224609375, 0.295654296875, 0.414306640625, -0.302734375, -0.5654296875, 0.36376953125, -0.9248046875, -0.0955810546875, -2.28515625, -0.54638671875, -0.92724609375, -0.79443359375, -0.320068359375, -0.58837890625, 1.4423828125, -0.3642578125, -0.02252197265625, -0.294921875, -0.69287109375, -1.5380859375, -0.77490234375, -1.7451171875, 0.231201171875, -1.2890625, -1.3310546875, 0.69384765625, -1.423828125, 1.5166015625, -0.515625, -0.720703125, -0.0086517333984375, -0.88671875, 0.5556640625, 0.96044921875, 0.84423828125, 1.5244140625, 0.332763671875, 1.0048828125, -1.0458984375, -0.0202484130859375, -1.23046875, -0.7177734375, 0.29150390625, -0.4453125, 0.3251953125, -0.172607421875, -0.5126953125, 0.005359649658203125, -0.6728515625, 0.366455078125, 2.70703125, 0.6396484375, 0.1593017578125, 1.900390625, 0.7978515625, -1.7509765625, 0.6259765625, -1.556640625, -1.4013671875, 1.39453125, -1.0244140625, -0.288330078125, -0.642578125, 0.93115234375, -0.0187835693359375, 0.489501953125, 0.137451171875, -0.3994140625, 0.320556640625, 0.58740234375, 0.78857421875, -0.253662109375, 0.2288818359375, -0.17724609375, -0.5126953125, -0.005939483642578125, -0.0012445449829101562, -0.65283203125, 0.0162811279296875, 0.59033203125, -0.728515625, 0.5732421875, 0.61962890625, -1.6484375, -0.062225341796875, 0.759765625, -0.875, -0.24755859375, -0.56201171875, 0.45458984375, 0.32958984375, 0.1881103515625, -1.330078125, -0.096435546875, 0.079345703125, -0.85009765625, -0.642578125, 0.153076171875, -2.216796875, 1.8212890625, 0.216796875, -0.72998046875, -0.156494140625, 2.955078125, 0.1800537109375, -1.298828125, 1.291015625, -1.3115234375, 1.400390625, -0.796875, 0.630859375, -0.5400390625, 0.11090087890625, -0.279052734375, 1.69140625, 1.3310546875, -1.92578125, 0.410400390625, -0.83349609375, -0.70263671875, 0.0208892822265625, -1.2734375, -0.1822509765625, -1.7822265625, 0.282958984375, -0.28173828125, 0.265869140625, -0.5732421875, 2.974609375, 0.410400390625, 0.455078125, -0.6083984375, 1.380859375, 1.4814453125, 1.755859375, -0.75634765625, 0.023712158203125, 0.2171630859375, -0.8701171875, 1.4111328125, -1.0615234375, -0.388916015625, 1.2939453125, -1.48828125, 0.81884765625, 0.315673828125, -3.669921875, -1.1181640625, -0.217529296875, -1.6171875, 0.30224609375, 0.58056640625, 1.3427734375, -0.40966796875, 0.6796875, -0.285888671875, 0.23291015625, 0.53759765625, 0.369140625, -0.2080078125, 0.88037109375, 0.1566162109375, -1.8896484375, 0.1754150390625, 0.9267578125, -0.4130859375, 0.016876220703125, 0.27783203125, 0.61962890625, -1.326171875, 1.1064453125, 0.255615234375, 1.3984375, -0.79541015625, 0.12139892578125, 0.8779296875, 1.8212890625, -0.1295166015625, 0.360595703125, -0.67919921875, 0.45166015625, -1.0302734375, -0.73681640625, 0.64453125, -0.62646484375, -0.5537109375, 0.7001953125, -1.7763671875, 0.73876953125, 1.2255859375, -0.1746826171875, 0.71240234375, -3.478515625, -0.5029296875, -1.146484375, -0.1268310546875, -0.091796875, 0.58056640625, -0.467041015625, 0.6376953125, -0.3203125, 0.77880859375, 0.76513671875, -0.9208984375, -1.6025390625, -0.057769775390625, 1.318359375, 0.87646484375, -0.258056640625, 0.05438232421875, 0.81640625, -1.6875, -0.8994140625, 0.7109375, 0.6435546875, -1.447265625, 0.2484130859375, -0.0298919677734375, -0.68017578125, -0.89697265625, 1.296875, -0.24365234375, -0.6865234375, -1.55078125, 2.837890625, -0.247802734375, 0.58837890625, -1.19921875, -0.63134765625, -0.0234527587890625, -1.3876953125, 1.978515625, -0.59716796875, -0.74169921875, 0.7451171875, 0.0220489501953125, 0.093017578125, 1.0771484375, 0.187744140625, 0.2763671875, 1.0341796875, -0.54443359375, -0.67138671875, 0.0716552734375, 0.230224609375, -1.119140625, -0.33984375, 2.279296875, -0.8388671875, 0.0528564453125, -2.634765625, 0.70068359375, 1.2646484375, -0.7080078125, 0.0880126953125, 0.247802734375, -0.8544921875, -0.1890869140625, -0.67138671875, 1.380859375, -1.0849609375, -0.44189453125, -0.18359375, 0.32958984375, 1.130859375, 0.65087890625, 1.7646484375, -0.61328125, -1.27734375, 0.1483154296875, 0.2200927734375, 1.16796875, -0.64697265625, -0.107666015625, 1.25390625, 0.58984375, 1.8466796875, -0.95703125, -1.1435546875, -0.69189453125, -0.587890625, 1.9482421875, 0.32373046875, 0.82373046875, -0.76025390625, -0.529296875, 0.1287841796875, -1.2373046875, 0.69140625, 1.123046875, -0.26025390625, -0.66796875, -0.43994140625, -0.9951171875, -2.314453125, -1.7646484375, 0.73095703125, -0.7861328125, 0.1798095703125, -0.2274169921875, -1.376953125, 0.472900390625, -1.6279296875, -2.81640625, -0.84716796875, -0.54443359375, -2.861328125, 0.6455078125, -0.445068359375, 0.52099609375, 1.2666015625, -1.3779296875, 0.401123046875, -0.515625, 1.037109375, -0.68701171875, 0.493896484375, 0.06378173828125, 0.156982421875, 0.57177734375, 0.1783447265625, -1.466796875, 1.232421875, 0.474853515625, -0.93603515625, 0.90380859375, -2.990234375, -0.453857421875, -0.324462890625, -1.6015625, -0.71337890625, 1.5146484375, -0.5068359375, 0.65771484375, -0.8916015625, -0.78369140625, 0.173095703125, 0.07470703125, 2.205078125, 0.239013671875, -1.1943359375, -0.2208251953125, 1.1611328125, 0.448974609375, 0.09466552734375, 0.865234375, -0.76611328125, 0.6845703125, -0.2008056640625, -1.2294921875, -0.81884765625, -1.3154296875, -0.8466796875, -0.7666015625, -0.36474609375, 0.79931640625, -0.425537109375, -0.7001953125, 1.2470703125, -1.373046875, 2.353515625, -0.2332763671875, 0.65185546875, -1.2109375, -0.86279296875, -0.33251953125, -1.015625, 1.9833984375, 0.8291015625, 0.58447265625, -0.477783203125, -0.265869140625, 1.28125, 2.17578125, 1.4033203125, 0.61328125, 0.336669921875, -0.56591796875, 0.92431640625, 1.37890625, 0.00785064697265625, -0.344482421875, 0.89892578125, -0.70703125, -1.599609375, 0.52099609375, 0.89013671875, 1.1298828125, -1.6123046875, 1.03125, -0.86279296875, 0.62451171875, 1.7509765625, 1.0009765625, -0.484619140625, -1.6767578125, -0.18896484375, -0.13916015625, -0.331787109375, 1.0302734375, -0.412109375, 1.318359375, -1.3310546875, -0.2113037109375, 0.465576171875, 1.0498046875, -1.21484375, -0.84130859375, 1.240234375, -0.365966796875, -1.2578125, -0.87548828125, -0.5068359375, -0.13720703125, 0.87255859375, 2.283203125, 1.3466796875, -0.2344970703125, -0.7822265625, -0.08251953125, 0.36474609375, 2.3515625, -0.70458984375, 0.5126953125, -0.441650390625, 0.01904296875, 0.4609375, 0.027923583984375, -1.853515625, 0.3203125, 0.89013671875, 1.125, 1.5595703125, -0.437744140625, 0.59033203125, -0.0183258056640625, 0.7822265625, -2.96484375, -0.85302734375, -0.56005859375, 0.414306640625, 1.2958984375, 0.740234375, 0.92822265625, 0.468505859375, -1.8681640625, -0.18017578125, -0.494140625, -0.08770751953125, -1.3369140625, -1.53125, 0.60205078125, -0.75537109375, 2.43359375, 0.73046875, -1.24609375, -0.2548828125, -0.53955078125, -0.287841796875, 1.3525390625, -2.27734375, -2.208984375, 0.259765625, -2.140625, 1.572265625, -0.498046875, 0.48291015625, 1.0634765625, 1.154296875, -0.5263671875, -1.560546875, -0.433349609375, -0.067138671875, 0.50927734375, -0.46923828125, 0.6005859375, 1.2021484375, 0.900390625, -1.4921875, -0.69775390625, 0.28955078125, -0.437255859375, -1.1728515625, 0.45263671875, -1.2080078125, -0.032989501953125, -0.57861328125, 0.52880859375, 0.9423828125, -1.1552734375, 0.599609375, -0.994140625, -0.33154296875, 0.68603515625, -0.796875, 1.427734375, 0.046234130859375, 0.8466796875, 0.89111328125, -0.34912109375, 0.58349609375, -0.60595703125, 0.73583984375, -0.58056640625, -0.375732421875, 0.4267578125, -1.1220703125, 1.591796875, 0.283935546875, 1.52734375, 0.8193359375, 2.158203125, 0.39794921875, 1.03515625, -0.64599609375, 1.08203125, -2.1484375, -1.0009765625, -2.142578125, -0.08978271484375, -0.392333984375, 1.546875, -1.4140625, 1.146484375, -0.806640625, 1.3720703125, 0.1226806640625, -0.44921875, 1.1884765625, 0.88525390625, -1.0234375, 0.09356689453125, 1.916015625, -0.39990234375, -0.42431640625, 0.4755859375, 0.267333984375, 1.03515625, 0.78173828125, -0.5068359375, 0.50634765625, -0.63720703125, -0.258056640625, -0.259521484375, -1.0146484375, -0.54443359375, 1.212890625, -1.45703125, -1.8896484375, -2.43359375, 0.728515625, 1.369140625, -0.01493072509765625, -0.344970703125, -1.5859375, -0.4833984375, 0.72900390625, -0.2152099609375, 0.06298828125, -0.09613037109375, -0.96044921875, 1.044921875, -1.0078125, 1.4873046875, -0.061431884765625, 0.61328125, 0.436767578125, -2.322265625, -0.69140625, 0.8251953125, -0.125244140625, 0.0865478515625, 0.40478515625, 1.9931640625, -0.281005859375, -0.6279296875, -0.51513671875, -0.064208984375, -0.454833984375, -0.031280517578125, -0.252197265625, -1.060546875, 0.71533203125, 0.162841796875, -0.1568603515625, 0.9443359375, 0.87158203125, 0.5087890625, -1.236328125, -1.123046875, -1.21875, -0.8876953125, 0.1263427734375, -0.433837890625, 0.3251953125, -0.6044921875, -0.9013671875, -0.39794921875, 1.71875, -0.330322265625, -0.4853515625, 0.355224609375, 0.256591796875, 0.5625, 0.013641357421875, -0.458740234375, 0.112548828125, 0.8349609375, 0.455078125, -0.6513671875, 1.29296875, 0.414306640625, -1.271484375, 0.05615234375, 0.55517578125, 0.13330078125, -0.091552734375, -0.33154296875, -0.03302001953125, -0.4892578125, 0.335205078125, -0.0263214111328125, -0.344482421875, -0.09332275390625, 1.060546875, 1.826171875, 0.755859375, 1.138671875, 1.0361328125, 0.45556640625, -0.2293701171875, 1.3828125, -0.93408203125, -0.164794921875, -0.05572509765625, 1.0107421875, 1.326171875, -0.9033203125, -0.421875, -1.2978515625, -0.6513671875, 0.515625, -1.12890625, 0.062103271484375, 1.046875, -0.2042236328125, 0.51025390625, 0.52880859375, 1.5986328125, 0.67236328125, -1.0224609375, -0.05633544921875, 0.4599609375, -0.59375, -0.67626953125, -1.4091796875, 0.6650390625, -0.24267578125, -0.481201171875, 0.94384765625, 1.2978515625, 1.0400390625, 1.5380859375, -1.32421875, 1.0634765625, -0.17626953125, 0.99853515625, 0.062347412109375, 1.2470703125, 0.0660400390625, 0.2587890625, -0.84765625, -0.306640625, -0.8369140625, 0.005344390869140625, 0.83056640625, 1.287109375, 0.485595703125, 0.06292724609375, -0.44287109375, 0.12890625, -2.326171875, 0.1300048828125, -0.27490234375, 0.435791015625, 0.340087890625, 0.349365234375, 0.03253173828125, -1.474609375, -0.2215576171875, -0.68017578125, 0.061248779296875, 0.9267578125, 0.1505126953125, -0.3623046875, 0.390625, -0.60400390625, -1.728515625, -1.3681640625, -0.2003173828125, -0.75341796875, 2.166015625, -0.671875, -0.9130859375, 0.45751953125, 1.3740234375, 0.08526611328125, 1.966796875, 0.34814453125, 0.765625, -1.62890625, -0.6083984375, 0.478759765625, -0.39697265625, -2.091796875, 0.354736328125, 0.2257080078125, -1.2158203125, 1.0283203125, 0.541015625, -2.193359375, -1.30078125, 0.77978515625, -0.08038330078125, -0.52978515625, 0.7783203125, 0.03399658203125, -0.267578125, -0.044677734375, 0.320556640625, 0.64501953125, -0.1929931640625, -0.3505859375, 2.890625, 1.318359375, -1.5859375, -1.3916015625, -0.39013671875, -1.873046875, -0.11065673828125, -0.02764892578125, -0.10064697265625, 0.462158203125, -0.092041015625, -1.2568359375, 0.93310546875, 0.149169921875, 1.60546875, -0.291015625, 0.87890625, 0.70361328125, -0.841796875, -0.94921875, 2.330078125, -1.84375, 0.10992431640625, 1.8271484375, 2.271484375, 0.78173828125, 1.615234375, 0.0011625289916992188, 0.2254638671875, 0.92919921875, 0.58642578125, -1.025390625, 1.3583984375, -0.66162109375, -1.5087890625, -1.2958984375, 0.130615234375, 0.17041015625, -1.1396484375, -1.90625, 0.3525390625, 0.689453125, -0.6328125, 0.89892578125, 0.1572265625, -0.4619140625, 0.642578125, 1.2412109375, -0.331298828125, 0.63623046875, 2.189453125, -0.2086181640625, 0.16845703125, -0.037445068359375, -0.37109375, -0.451416015625, 0.093994140625, -0.2308349609375, -0.7529296875, -0.80859375, 0.388916015625, -0.76611328125, 2.138671875, -1.412109375, -0.0389404296875, -0.349853515625, 0.311279296875, -2.580078125, -0.8046875, 0.1654052734375, 0.179443359375, -0.29443359375, 1.216796875, -0.8603515625, -0.66650390625, 1.2685546875, 0.307861328125, 1.20703125, -0.70556640625, -1.3173828125, -0.142578125, -0.79443359375, -0.12445068359375, 1.0390625, 1.2587890625, 0.76513671875, -0.09124755859375, 1.92578125, 1.9794921875, 1.2841796875, 0.177001953125, 0.7275390625, 0.333740234375, -0.93603515625, -0.154296875, 0.1114501953125, 0.72509765625, -1.3359375, -0.399658203125, 1.9423828125, 0.83740234375, 0.0226898193359375, 0.880859375, 0.57666015625, 0.1534423828125, 0.93359375, -0.72998046875, -1.0546875, -1.736328125, -0.412353515625, -1.2373046875, 0.68798828125, -1.2412109375, 0.336181640625, 1.2353515625, -0.978515625, -0.24267578125, 0.012847900390625, 0.794921875, 0.42578125, 1.2294921875, 0.712890625, 1.4638671875, -0.185546875, 0.4921875, -1.3798828125, 1.365234375, 0.759765625, 0.1318359375, 1.365234375, -0.84716796875, 1.07421875, 1.5673828125, 0.78759765625, 0.1695556640625, -1.02734375, 0.219482421875, 1.33984375, -0.325927734375, 0.472900390625, -0.921875, -0.2008056640625, -1.3203125, -0.3037109375, 0.5595703125, -0.931640625, -0.426513671875, -0.484619140625, -0.494384765625, -0.45166015625, -0.66552734375, 0.33447265625, 0.94091796875, -0.29150390625, 1.283203125, -0.53564453125, 0.302734375, -1.0986328125, -0.60009765625, 0.0096282958984375, -0.7080078125, -1.310546875, -1.0595703125, 1.1533203125, -0.241943359375, 0.41845703125, 0.083984375, -0.33056640625, -0.62353515625, -0.175537109375, -1.109375, -0.68408203125, 2.0625, 0.63427734375, -0.2783203125, 0.143798828125, -0.464599609375, 0.46142578125, 0.1805419921875, -0.35400390625, 1.0205078125, 0.2320556640625, 1.9619140625, -0.33349609375, 0.89794921875, -2.029296875, -1.06640625, 0.1820068359375, -0.330078125, 0.072021484375, -2.041015625, 0.472900390625, -1.3740234375, 1.375, 0.496337890625, 0.87060546875, 0.560546875, -0.42041015625, -3.017578125, 0.049835205078125, 0.94775390625, -0.412841796875, -1.4326171875, -0.650390625, -0.84228515625, 1.8916015625, -1.150390625, -1.5302734375, -0.53076171875, 0.1953125, 0.416015625, 0.282470703125, -0.60498046875, -1.896484375, -0.1256103515625, -0.056396484375, -0.5126953125, -0.73046875, 1.0908203125, 1.1474609375, 1.21875, -0.2276611328125, -0.8818359375, 0.443603515625, 0.5107421875, -1.0078125, 0.61962890625, 1.03515625, -0.369140625, -0.7119140625, 0.340576171875, -0.486083984375, -0.39208984375, -0.36279296875, -0.468017578125, -2.626953125, -0.52490234375, 1.3515625, -0.088134765625, 0.2186279296875, -0.327880859375, 0.5458984375, 0.94775390625, 2.296875, 0.385009765625, -0.44873046875, -0.04412841796875, -0.468994140625, -0.89697265625, -0.0333251953125, 0.38671875, -0.2384033203125, -0.2315673828125, 1.1533203125, 0.7373046875, -0.077880859375, 1.0615234375, 2.0859375, -1.4765625, 0.16650390625, -0.6123046875, 2.23046875, -0.51513671875, 1.0302734375, -0.7802734375, -0.150146484375, -0.5537109375, 0.60302734375, 1.45703125, -0.2047119140625, -1.1728515625, 0.68994140625, -1.3603515625, 0.3681640625, -0.3525390625, 0.1544189453125, -0.89404296875, -0.1085205078125, 1.37890625, -0.72119140625, 0.04693603515625, 0.411865234375, 0.499755859375, 0.06646728515625, 0.37890625, -0.0654296875, -0.3037109375, -0.36083984375, 1.2197265625, -1.5048828125, 0.84033203125, -1.28515625, -0.57568359375, 1.3681640625, -0.98828125, -1.3349609375, -1.177734375, -1.1044921875, 2.068359375, -1.19921875, -2.048828125, 0.041778564453125, -0.65869140625, -0.0240936279296875, 1.9306640625, 0.444580078125, -0.1705322265625, 0.5126953125, -0.07745361328125, -0.08892822265625, 1.0830078125, -0.5224609375, -0.4072265625, 0.1632080078125, 0.98095703125, 0.22705078125, 0.0241851806640625, 1.0673828125, -0.6240234375, 0.93603515625, 1.49609375, -2.115234375, 0.047027587890625, -1.515625, -1.4150390625, 1.0068359375, -1.7548828125, 2.203125, -1.2431640625, -0.8466796875, 0.74072265625, -0.394287109375, 0.044403076171875, -1.361328125, 2.41796875, 0.179443359375, -1.705078125, 0.9521484375, 1.302734375, -0.03289794921875, -1.7451171875, 0.302490234375, -0.970703125, 0.197265625, -1.375, 0.274169921875, -1.5361328125, -1.2578125, -0.5556640625, -0.85888671875, 1.58203125, -0.040771484375, -1.1533203125, -0.9892578125, -2.0546875, 0.12744140625, -1.0302734375, -0.904296875, 1.353515625, -1.3525390625, 0.810546875, 2.01171875, 1.1142578125, 1.67578125, -0.8896484375, -0.105224609375, 0.2255859375, 1.2470703125, 2.95703125, -0.119873046875, -0.159423828125, 1.8994140625, -0.348388671875, -0.053558349609375, -0.256591796875, -1.4326171875, -0.39111328125, 0.1875, -0.90478515625, 0.45458984375, 0.7353515625, 1.2421875, -0.31787109375, -0.095458984375, -0.45263671875, -1.3642578125, -2.263671875, 0.257568359375, 0.69189453125, -0.52001953125, 1.3662109375, -0.153564453125, 0.0310211181640625, -0.4697265625, -0.66748046875, 0.6005859375, -0.732421875, -1.365234375, 0.6650390625, 0.1947021484375, 0.5712890625, 0.04510498046875, 0.9638671875, -0.1195068359375, -0.244140625, -0.8037109375, -0.2357177734375, 0.419921875, -0.54296875, -1.328125, -0.41357421875, 0.2646484375, 0.1387939453125, -0.49853515625, -0.2548828125, 0.6376953125, 0.68017578125, 0.5791015625, 1.0791015625, 0.5224609375, -1.0322265625, 1.78125, -0.302490234375, -0.116455078125, -0.056915283203125, 0.876953125, -0.392333984375, -0.378173828125, 1.8369140625, 0.1981201171875, -0.76220703125, -1.1630859375, -0.68212890625, -0.06884765625, 0.327392578125, -1.021484375, 0.52880859375, -0.374755859375, 1.4931640625, 2.43359375, 0.775390625, -2.07421875, 1.6884765625, -1.677734375, -0.25634765625, -1.2255859375, -1.037109375, -0.7998046875, 0.89990234375, -1.123046875, 0.53271484375, -1.359375, 0.11737060546875, 0.24267578125, -1.3232421875, -0.3359375, -0.360107421875, 0.7099609375, -0.259521484375, 0.6337890625, 1.2392578125, -0.52783203125, 0.497314453125, 0.544921875, 0.56884765625, -1.236328125, 0.09014892578125, -0.0616455078125, 1.681640625, -0.081787109375, -0.386474609375, 0.7919921875, 2.544921875, 1.3583984375, -0.07928466796875, 0.73095703125, -0.9306640625, -0.576171875, 0.423095703125, -0.163818359375, -0.66064453125, -0.005977630615234375, 0.458251953125, 0.34375, 0.708984375, -2.16796875, -1.1845703125, -0.7607421875, -0.3857421875, 0.362060546875, 0.5322265625, -0.40380859375, -0.64404296875, 0.042572021484375, -1.361328125, -0.0026702880859375, -0.1591796875, 1.1767578125, 0.335205078125, -1.37109375, 1.0634765625, -1.130859375, 0.55712890625, -0.40869140625, 0.11676025390625, 0.23779296875, 0.40478515625, 2.392578125, 0.1734619140625, -0.62548828125, 1.1552734375, 0.307861328125, 0.89794921875, -1.216796875, -0.047271728515625, -0.2265625, 0.505859375, 0.77880859375, -1.54296875, -0.66455078125, 0.056793212890625, -0.24462890625, 1.7861328125, -0.54833984375, 1.0458984375, -1.0048828125, 1.7958984375, -0.1654052734375, -0.07562255859375, 1.8798828125, 0.5712890625, -0.93505859375, -1.642578125, -0.78564453125, 0.3505859375, -0.435791015625, -0.393798828125, -1.0654296875, 1.564453125, -0.293212890625, -0.273193359375, 1.9248046875, -1.427734375, -1.119140625, 1.033203125, -1.6103515625, -2.259765625, -0.951171875, 1.296875, -0.42138671875, 1.95703125, 0.3125, -1.39453125, -1.5751953125, -1.1904296875, -0.03900146484375, 1.9560546875, 1.0302734375, -0.7412109375, 0.8310546875, 0.8447265625, -1.0009765625, 1.009765625, 1.1435546875, -1.4384765625, -0.1280517578125, 0.3955078125, -0.93408203125, 1.7060546875, 0.1859130859375, -0.7373046875, 0.22900390625, -0.90673828125, -1.017578125, 0.260498046875, -0.407958984375, -1.337890625, 1.1005859375, -0.122802734375, -0.0140228271484375, -0.428466796875, 0.15771484375, 0.4052734375, -0.94091796875, 1.3017578125, 0.6015625, -0.6982421875, -0.736328125, 0.0692138671875, -0.63427734375, 1.650390625, 0.91357421875, -0.243408203125, -0.6640625, -0.68505859375, 2.078125, -0.5166015625, 0.14013671875, -0.279052734375, 0.1805419921875, -0.75244140625, 0.053741455078125, 0.217529296875, 0.3603515625, 0.8134765625, -0.2099609375, -0.0044403076171875, 0.1490478515625, 0.990234375, -0.11431884765625, -0.399169921875, 1.2978515625, 0.830078125, -0.08099365234375, 0.61572265625, 0.97412109375, 2.23046875, -1.4345703125, 0.60595703125, 2.259765625, -0.572265625, -0.71044921875, -1.4462890625, 0.8876953125, -0.49755859375, 0.36279296875, -0.489990234375, 0.343017578125, 0.98046875, -0.7236328125, 0.422119140625, -1.5791015625, -0.2235107421875, 0.7568359375, -0.407958984375, 1.8212890625, -1.720703125, -1.5654296875, -1.0634765625, -0.875, -0.8017578125, -0.67529296875, 0.80517578125, 0.342529296875, -0.7412109375, 1.3720703125, -1.080078125, 0.62548828125, 0.2430419921875, 0.1024169921875, 1.275390625, 0.4345703125, 0.53466796875, -0.6162109375, 0.89453125, 1.4765625, -0.7138671875, 0.027679443359375, -0.322265625, 0.400146484375, -1.1455078125, 0.12890625, -0.52099609375, 0.108154296875, 0.8271484375, -0.495361328125, 0.59130859375, -1.7978515625, -0.685546875, -1.2451171875, -0.150146484375, -0.0135345458984375, -0.09259033203125, -0.1995849609375, 0.330078125, -0.265869140625, -0.6494140625, -1.009765625, 1.578125, -0.16455078125, -0.09930419921875, -0.1663818359375, -0.166259765625, -3.1796875, -0.1925048828125, 2.70703125, 0.469970703125, -0.28515625, -0.791015625, 0.295166015625, 0.65185546875, 0.4091796875, -0.295654296875, 0.8232421875, 1.43359375, 2.025390625, -0.43505859375, 0.587890625, 1.5830078125, -0.943359375, -1.5283203125, 0.59033203125, -2.033203125, -0.214111328125, -1.0712890625, -0.61181640625, -1.181640625, 1.669921875, 2.09375, 0.025726318359375, 1.6171875, -0.669921875, -1.2392578125, 0.24365234375, -0.174560546875, -1.8046875, -1.1376953125, 1.1669921875, 0.323486328125, 0.68310546875, 0.0272064208984375, 0.0219573974609375, -0.9580078125, 0.2401123046875, -0.6318359375, 0.51513671875, -0.5625, 0.45166015625, -0.403564453125, -1.7939453125, -1.140625, -0.90625, -0.99169921875, 0.2066650390625, 0.11712646484375, -2.279296875, -0.6015625, -0.2091064453125, -0.08526611328125, -0.5654296875, -0.7666015625, -0.548828125, -0.2406005859375, 1.896484375, 0.10992431640625, 0.5517578125, 0.32861328125, -0.2301025390625, -0.900390625, 0.205810546875, -0.65283203125, -0.26123046875, -0.2493896484375, 1.09375, -0.81884765625, -0.289794921875, 0.51513671875, -0.37744140625, -0.498779296875, 0.349609375, 0.755859375, -0.9990234375, -1.763671875, 0.7333984375, 0.66943359375, -0.146240234375, -0.68896484375, 0.38720703125, -0.049835205078125, 0.0006265640258789062, -0.97607421875, 1.3203125, 0.11053466796875, 1.5068359375, 0.458740234375, -1.162109375, 0.349365234375, 1.5869140625, -0.99609375, 0.027587890625, 0.5146484375, -0.350830078125, -0.400146484375, -0.6484375, -1.0263671875, -0.1531982421875, -0.1080322265625, -0.71826171875, 0.658203125, 0.46142578125, 0.90673828125, -0.99755859375, 0.587890625, -0.5322265625, 0.8740234375, 0.2027587890625, 1.3330078125, -0.354736328125, -0.321533203125, -1.0703125, 1.8369140625, -0.07928466796875, 0.1641845703125, 1.505859375, -1.06640625, -0.191650390625, 0.640625, -1.294921875, 0.1513671875, 1.3173828125, -0.31884765625, -0.2015380859375, -1.1298828125, -0.55615234375, -0.1116943359375, 0.0276641845703125, 0.5244140625, -0.052459716796875, -0.288330078125, 1.3203125, -0.267578125, -0.68798828125, 0.39013671875, 0.469482421875, -0.74072265625, -0.2200927734375, -0.5966796875, 0.537109375, 0.255615234375, 1.6259765625, -0.35693359375, -0.251953125, -0.1781005859375, 0.6513671875, -0.28076171875, 2.052734375, -3.029296875, -1.2626953125, -0.455810546875, 0.293212890625, 0.3408203125, -0.460205078125, 1.2080078125, 0.139404296875, -2.3203125, 1.0654296875, 0.163818359375, 0.845703125, -0.05584716796875, -0.76513671875, -1.685546875, -0.1414794921875, -0.736328125, -0.0233612060546875, -1.1357421875, -0.07696533203125, 0.0762939453125, 0.51123046875, 0.1478271484375, 0.345458984375, -0.32080078125, 1.126953125, 0.6591796875, 0.12432861328125, -1.15234375, -2.044921875, -1.7431640625, 0.7490234375, -0.0269012451171875, -1.0087890625, -0.74755859375, 1.11328125, -0.0034275054931640625, -0.36572265625, 1.8369140625, 0.79541015625, 0.06158447265625, -1.607421875, 1.9072265625, -1.4345703125, -0.91455078125, -0.9951171875, 1.5712890625, 0.94873046875, -0.003749847412109375, -0.399658203125, -1.0966796875, 0.22314453125, -2.068359375, 1.3408203125, -0.541015625, -1.6474609375, -1.3828125, -0.66259765625, -0.1527099609375, -0.453125, -1.55859375, -0.312255859375, 2.537109375, 0.93798828125, -1.1103515625, 0.10894775390625, 0.51416015625, -0.5537109375, 0.427490234375, -0.01528167724609375, -0.65087890625, 0.57470703125, -2.423828125, -0.81298828125, 1.4599609375, -1.6767578125, -1.4248046875, -0.591796875, -1.5654296875, -0.0241241455078125, -0.3193359375, -0.07794189453125, -0.11669921875, 0.08770751953125, 0.80712890625, 1.1953125, 1.3291015625, -0.489013671875, -1.087890625, 0.45849609375, 0.17431640625, 0.64697265625, -0.80126953125, -0.302734375, 0.9765625, -1.212890625, -0.095458984375, 0.41357421875, -0.72412109375, -0.458984375, -1.1787109375, -0.525390625, -0.7421875, -0.54150390625, -0.98095703125, -1.09765625, -2.056640625, -0.281982421875, -1.0205078125, 2.568359375, -0.6025390625, 1.8076171875, 0.314453125, 0.061004638671875, 0.216064453125, 0.385009765625, 0.70068359375, -0.59765625, -0.69677734375, 0.483154296875, 0.352783203125, -0.3662109375, -0.12548828125, 1.09375, -0.237060546875, 0.5576171875, -0.257568359375, 0.468017578125, 1.2763671875, 0.58935546875, 0.0599365234375, 0.38818359375, -0.35400390625, 0.2174072265625, 0.58935546875, 0.5966796875, -0.74951171875, 0.09063720703125, 0.0276641845703125, 1.1494140625, 0.57275390625, -0.002918243408203125, -0.8564453125, -0.263671875, -0.261962890625, -0.1785888671875, -0.55810546875, 0.450439453125, -1.6767578125, -1.27734375, -0.051788330078125, 0.95703125, -0.921875, 0.282470703125, 1.4814453125, -0.46533203125, -0.28759765625, -0.33740234375, 0.572265625, 0.015777587890625, 0.069580078125, -1.501953125, -0.42236328125, -0.260498046875, -0.75390625, 0.78125, -1.4990234375, -0.98193359375, 0.377197265625, -0.491455078125, -0.298583984375, 0.5830078125, 0.33544921875, -0.74267578125, 0.79052734375, 0.179931640625, -0.01371002197265625, -0.149169921875, 1.3232421875, -1.259765625, 1.01953125, -1.0703125, -0.9150390625, -1.59375, 0.55029296875, 0.11505126953125, -0.328369140625, 0.71630859375, 0.873046875, -1.490234375, 0.5732421875, -0.00621795654296875, -1.7431640625, -0.9619140625, 0.5478515625, 0.30322265625, -0.67138671875, 1.05078125, -0.361083984375, 0.32861328125, 0.63525390625, -1.57421875, 0.99658203125, -0.0175933837890625, 0.55859375, -0.09234619140625, -0.460205078125, 1.271484375, 1.193359375, -0.322021484375, -2.013671875, 0.425048828125, -0.35986328125, -0.281494140625, 0.251220703125, -1.203125, -1.6328125, -1.291015625, -1.2578125, 0.71240234375, 0.9404296875, -0.08721923828125, -2.001953125, -0.10699462890625, 1.4521484375, -1.4619140625, 0.794921875, -0.059906005859375, -0.32177734375, -0.74755859375, 0.63720703125, 0.8310546875, -0.1141357421875, 1.2177734375, 0.919921875, -0.79833984375, 1.6455078125, -0.316650390625, -1.798828125, -0.419921875, -0.1318359375, -0.9453125, -0.7734375, -0.72705078125, 0.09539794921875, -1.568359375, -3.34375, 0.56201171875, 0.68408203125, 0.427734375, -0.54931640625, 0.93017578125, 0.67822265625, 1.166015625, 1.099609375, -0.8662109375, -0.9287109375, 1.880859375, -0.71630859375, -0.362060546875, -0.2261962890625, 1.0263671875, -0.2471923828125, 0.03839111328125, -0.90966796875, 0.082275390625, 0.76220703125, 0.52880859375, 0.1597900390625, -0.192138671875, -0.32666015625, 0.4912109375, -1.1220703125, -0.1397705078125, 0.95654296875, -0.630859375, 0.7431640625, 0.87548828125, 0.5546875, -0.935546875, -1.779296875, -0.29443359375, 1.3798828125, -0.47119140625, -0.5302734375, 0.255859375, -0.98828125, -0.474365234375, 0.58544921875, -0.94921875, -1.287109375, 1.2265625, 0.6396484375, -0.459716796875, -1.921875, -0.75927734375, 0.410400390625, -1.1748046875, -0.392333984375, -0.485107421875, 1.435546875, 0.2161865234375, -1.40234375, -0.587890625, -0.65625, -0.73193359375, -0.1968994140625, 0.1610107421875, -3.08984375, -1.189453125, 0.08197021484375, 1.083984375, 0.308837890625, 0.71923828125, 0.51123046875, -1.259765625, 0.323974609375, 0.01007843017578125, 0.492919921875, 0.12225341796875, 0.56494140625, -0.491943359375, -0.484130859375, 0.01788330078125, 0.55224609375, -1.4443359375, 0.232666015625, -0.48779296875, 0.1474609375, 1.1689453125, 0.1986083984375, 0.31640625, -0.474365234375, -0.5380859375, 1.5341796875, 0.2412109375, 0.249267578125, 0.337890625, -0.8447265625, 0.1826171875, 0.1513671875, 1.6279296875, -0.0577392578125, -1.4375, -0.73681640625, 1.2802734375, -1.197265625, -0.65966796875, 0.56787109375, 0.135986328125, 2.228515625, -1.50390625, -1.0087890625, -0.2197265625, 1.890625, -0.23876953125, 0.60302734375, -0.197509765625, 1.0390625, 0.361083984375, 1.4638671875, -0.341796875, -0.435302734375, 1.5888671875, -0.1507568359375, -1.9541015625, 1.9150390625, 0.79541015625, -0.73681640625, -0.75048828125, -0.68115234375, 0.54248046875, -0.71533203125, 1.2861328125, 0.0936279296875, -1.1298828125, 0.8828125, -1.486328125, -0.091796875, 0.146240234375, 1.5009765625, -0.67529296875, -1.0712890625, -0.9736328125, -0.705078125, 0.305419921875, -0.501953125, 0.923828125, -1.052734375, 0.09283447265625, 2.275390625, -0.39501953125, -0.047882080078125, 1.4365234375, 1.4052734375, 0.309814453125, 0.471435546875, -0.0020618438720703125, -0.380859375, -0.39208984375, 0.2236328125, -0.93408203125, 0.50390625, -0.51806640625, -1.115234375, 0.7119140625, 0.079833984375, -1.6923828125, 1.1357421875, -2.271484375, -0.810546875, 1.2666015625, -0.5322265625, 0.2313232421875, 0.244873046875, -0.82373046875, 0.0640869140625, 1.208984375, 1.4580078125, 0.435302734375, -0.7392578125, -0.40380859375, 1.46875, 2.142578125, -1.0810546875, 0.5205078125, -0.58056640625, -1.15625, 1.5107421875, 0.10162353515625, 1.06640625, -1.00390625, -0.87353515625, 0.51220703125, 1.703125, 1.7001953125, 1.0205078125, -0.75732421875, 0.740234375, -1.0048828125, 0.038543701171875, -1.6298828125, -1.478515625, -0.84765625, 0.66845703125, -1.4599609375, 1.634765625, -0.182373046875, 2.126953125, 1.8232421875, 1.65625, -0.7314453125, -0.8798828125, -1.005859375, 2.123046875, -0.67431640625, 1.1484375, -0.2451171875, -1.6884765625, -1.4111328125, -0.5859375, -1.2392578125, 0.904296875, 1.689453125, 2.3203125, 0.387451171875, 0.11639404296875, -0.63525390625, 0.3046875, 0.1068115234375, -0.9736328125, -0.423828125, -1.3779296875, 0.497314453125, 0.671875, -3.052734375, 1.1962890625, 0.13037109375, 0.379638671875, 0.189453125, -2.16015625, -2.365234375, -0.426025390625, 0.55908203125, 1.087890625, -0.85595703125, -1.0732421875, 0.990234375, 0.269775390625, -0.52734375, -0.93701171875, -1.1826171875, -0.206787109375, -0.11065673828125, -1.8740234375, 0.650390625, 0.337646484375, 0.026092529296875, -1.0341796875, 1.2607421875, 0.70361328125, -0.48095703125, -0.03924560546875, -3.48828125, 1.8740234375, -0.75732421875, -0.206787109375, -1.3271484375, 0.7734375, 0.896484375, 1.025390625, 0.203369140625, 0.4208984375, 1.7177734375, 1.1884765625, -0.318115234375, -0.701171875, -0.15478515625, -0.06689453125, 0.36865234375, -1.28125, 0.30224609375, -2.28125, -0.281494140625, 0.13818359375, 1.8369140625, 2.12890625, 0.78466796875, -0.1646728515625, 0.84033203125, -0.080322265625, -0.99658203125, -0.609375, 1.3623046875, 0.6123046875, -0.71337890625, 0.6630859375, 0.7060546875, 1.7783203125, 0.351806640625, 2.009765625, -0.68212890625, -0.6943359375, -0.580078125, -0.65087890625, -0.94775390625, 0.3359375, -0.1484375, -1.6513671875, -0.33154296875, 0.71728515625, -0.91650390625, -0.2364501953125, -1.216796875, -0.453369140625, 2.716796875, 0.2099609375, 0.2322998046875, -0.337890625, -0.13818359375, -2.94921875, 0.439208984375, -0.41162109375, -0.2281494140625, 0.75927734375, -0.5224609375, 0.95654296875, -0.83935546875, -0.444580078125, 0.90673828125, 0.88232421875, -0.1580810546875, 0.413330078125, 2.53125, -0.40087890625, 0.55908203125, -1.076171875, -0.020172119140625, 1.0966796875, -1.345703125, -0.6572265625, 1.2705078125, -2.642578125, 0.326904296875, 0.58056640625, 0.35986328125, 0.1470947265625, 0.70751953125, 0.0926513671875, 0.81591796875, 2.123046875, 0.66552734375, -0.10260009765625, 0.21630859375, 0.42724609375, -1.025390625, -0.52294921875, -0.54541015625, -1.5283203125, -0.7158203125, -0.31884765625, 1.8388671875, -0.274658203125, -0.293701171875, -0.73583984375, -0.305419921875, -1.2841796875, 0.61279296875, 0.471435546875, -1.6123046875, -0.2193603515625, 0.456298828125, 0.343994140625, -0.0250244140625, 1.263671875, -0.66650390625, 2.447265625, 1.6435546875, 0.62548828125, -1.1162109375, 0.1773681640625, 1.8544921875, 2.060546875, 2.33203125, 0.65625, 1.1962890625, -1.337890625, -0.405517578125, -0.427734375, -1.1201171875, -1.259765625, -1.1181640625, -0.5400390625, -1.20703125, 0.83740234375, -0.177490234375, 0.1826171875, 0.1466064453125, -1.8486328125, 1.638671875, -0.568359375, 0.132568359375, 1.1064453125, 0.96044921875, -0.453857421875, -1.6943359375, 0.12646484375, 0.10247802734375, 0.9462890625, -0.05438232421875, 1.484375, 0.54052734375, 0.404296875, 0.830078125, -0.219970703125, 0.25732421875, -0.36328125, 0.057403564453125, -0.56640625, -0.26513671875, -1.3681640625, 0.06707763671875, -0.2388916015625, 0.90380859375, 0.64599609375, -0.8740234375, -0.405517578125, 1.451171875, -0.5126953125, 0.54833984375, 0.56201171875, 1.310546875, -0.033782958984375, -0.767578125, 0.78125, 1.9814453125, 0.927734375, -0.56396484375, -1.1875, 0.771484375, -0.258544921875, 0.55029296875, 0.52197265625, -0.7626953125, 0.373779296875, -2.201171875, 0.06689453125, 1.23828125, -0.69775390625, -1.791015625, -0.383544921875, -2.0859375, -1.69140625, -0.11834716796875, 0.0279388427734375, -1.4970703125, -0.30078125, 1.205078125, 1.2119140625, -1.3203125, 2.357421875, -1.3935546875, -1.236328125, -0.67578125, -1.8603515625, -0.50634765625, -0.5576171875, 0.6943359375, 0.4091796875, 0.01311492919921875, 0.007656097412109375, 0.73193359375, -0.172607421875, 0.058502197265625, 0.65869140625, -1.2548828125, -0.59619140625, 2.484375, -0.5654296875, -0.8203125, -1.4169921875, 1.69921875, -0.796875, 1.271484375, -1.2861328125, 1.59375, -1.259765625, 0.499755859375, 0.6884765625, 0.97412109375, -0.1514892578125, -0.89794921875, 0.341064453125, 1.759765625, -0.2232666015625, 1.7626953125, -0.67822265625, 0.4931640625, 0.95947265625, 0.1441650390625, -1.1630859375, -0.228271484375, 0.480224609375, 0.322021484375, 1.37109375, -1.0146484375, 0.217041015625, 0.62255859375, 1.5927734375, -0.421630859375, 1.294921875, -0.65380859375, -0.77783203125, 0.7763671875, 1.865234375, 1.6318359375, -1.4306640625, -0.64111328125, -1.021484375, -0.82177734375, -1.1328125, 1.09765625, -0.48583984375, -0.884765625, 0.87841796875, -0.66943359375, 2.69921875, 0.1329345703125, -0.1082763671875, 1.2119140625, -0.50732421875, -1.5048828125, 0.471923828125, -0.342041015625, -0.94140625, 0.1507568359375, -1.2548828125, -0.11956787109375, 0.76416015625, -0.25537109375, 0.13427734375, -2.412109375, -1.1728515625, 0.339111328125, 2.11328125, -0.5048828125, -1.0771484375, 2.263671875, 2.09375, 0.3408203125, 0.52587890625, -0.410888671875, -0.42529296875, -0.0379638671875, 0.65673828125, -2.369140625, -1.3095703125, 1.1650390625, 0.360107421875, 2.052734375, -1.2841796875, -0.6044921875, -1.8994140625, -0.69287109375, 0.74072265625, 1.0478515625, -1.3310546875, -0.97607421875, -0.190673828125, -0.318603515625, 1.064453125, 0.289794921875, -0.50048828125, 1.5029296875, -0.1873779296875, 0.065185546875, 1.4658203125, -0.55419921875, 0.448486328125, -1.1962890625, -0.0836181640625, -1.0751953125, -0.278564453125, 0.36767578125, 0.1588134765625, 1.5322265625, 0.337158203125, -1.30078125, 0.033477783203125, 1.3642578125, -0.0184173583984375, -1.0234375, -0.80810546875, 0.489501953125, -0.8505859375, 0.578125, -1.201171875, 0.5419921875, 0.3076171875, 1.28125, 0.30322265625, 1.623046875, 0.94873046875, -0.470947265625, -1.970703125, -0.66650390625, 1.880859375, -1.0673828125, -1.486328125, -0.92578125, 0.277099609375, -0.1588134765625, 1.0107421875, -0.5146484375, 0.2479248046875, -0.321044921875, 0.2022705078125, 1.919921875, 0.3662109375, -0.65966796875, -0.83544921875, 2.052734375, 1.52734375, -0.488037109375, -1.7890625, -1.1962890625, 0.0963134765625, 1.380859375, -1.3037109375, 0.78759765625, -2.302734375, -0.9326171875, 3.28125, 1.6513671875, -1.1796875, -0.86474609375, -0.7451171875, -0.490478515625, 0.313720703125, 0.03204345703125, -0.5283203125, 0.814453125, -0.45556640625, 1.14453125, -0.339599609375, -1.0380859375, 0.875, -1.2099609375, -0.95703125, -1.556640625, 0.489501953125, -0.8203125, -0.681640625, -0.49609375, 1.76953125, -0.241943359375, -0.65869140625, -1.1630859375, -0.17431640625, -0.46142578125, 0.89501953125, 1.59375, 0.5283203125, -2.4453125, -0.9052734375, -0.1280517578125, -0.0548095703125, 0.229736328125, -0.65625, -0.477783203125, -0.49072265625, -1.166015625, -1.0849609375, -0.0047454833984375, -0.13671875, 0.67236328125, 0.46044921875, 0.297119140625, -0.6220703125, 1.9541015625, 0.1270751953125, 1.638671875, 0.175537109375, 1.662109375, -0.28564453125, 1.0830078125, 1.2119140625, 0.7666015625, -0.72998046875, -0.78857421875, -1.8974609375, -0.2392578125, -0.443115234375, -1.138671875, 0.62353515625, -1.9228515625, -1.4697265625, 1.18359375, 1.0322265625, 0.39208984375, 0.80029296875, -0.115234375, 1.05859375, -0.58740234375, -0.1507568359375, -0.82666015625, 1.9931640625, -0.6533203125, -0.1473388671875, -0.43505859375, 0.492919921875, 0.17919921875, 0.046173095703125, -0.5927734375, 0.37646484375, -1.5947265625, -0.1981201171875, -1.6083984375, 1.361328125, 0.64501953125, 0.37255859375, 0.5390625, 0.69775390625, 1.3046875, 0.09442138671875, 0.5302734375, -0.24560546875, 0.63720703125, 1.216796875, -1.697265625, -0.11456298828125, -1.203125, 0.3046875, 1.5849609375, -0.1585693359375, 1.5498046875, 1.2138671875, 0.365478515625, 1.126953125, -1.0107421875, -0.11456298828125, -1.4521484375, 0.368896484375, -0.2064208984375, -2.287109375, 0.0126953125, -0.2391357421875, -0.43017578125, 0.01068115234375, -0.1317138671875, 0.83154296875, 1.47265625, 1.15625, -0.298583984375, -0.354736328125, -0.367431640625, 0.73828125, -0.264404296875, -1.3447265625, -0.97314453125, -2.201171875, -2.03515625, 0.308349609375, 1.123046875, 1.9560546875, 1.1787109375, 1.5283203125, 1.3271484375, 1.1669921875, -1.1015625, 0.7724609375, -0.8837890625, 0.3955078125, -1.0478515625, -1.9326171875, -0.91259765625, -1.31640625, -0.517578125, -1.140625, -0.274169921875, 1.4248046875, 0.053497314453125, -0.97412109375, -1.1826171875, -1.275390625, -0.49072265625, 0.1092529296875, -0.77392578125, -1.0439453125, -0.66650390625, -1.296875, -1.7080078125, -0.619140625, -1.201171875, -0.2235107421875, -1.6083984375, -1.595703125, -0.40576171875, 1.2880859375, -1.0517578125, -0.7705078125, -0.63427734375, -1.6943359375, -0.771484375, -0.70556640625, 0.8271484375, -0.92919921875, -1.5908203125, 1.0322265625, 1.103515625, -1.533203125, 0.4130859375, -0.54345703125, 0.28564453125, 0.5732421875, -0.493408203125, -0.6943359375, -0.159423828125, 0.87939453125, -0.6533203125, 0.70654296875, -1.283203125, 0.188232421875, -0.04486083984375, -1.263671875, 0.06195068359375, 0.483642578125, -1.3349609375, -0.354736328125, -0.93798828125, -2.826171875, -0.80517578125, 2.04296875, -0.272705078125, 1.7607421875, -0.68603515625, -0.8037109375, 0.08843994140625, -0.13671875, 0.032806396484375, 0.0011148452758789062, -0.626953125, 0.50048828125, 0.11279296875, -0.87255859375, -0.6240234375, 0.69873046875, 1.3466796875, -0.264404296875, -1.1064453125, -0.92822265625, 2.0703125, -0.265625, 0.50634765625, 0.49462890625, 2.734375, 0.0838623046875, -0.69677734375, -0.62255859375, -0.810546875, -0.194091796875, 0.935546875, 0.5205078125, 0.425537109375, 0.65673828125, -1.154296875, -0.9052734375, 0.86376953125, 1.0029296875, -1.2607421875, -1.3896484375, -0.392822265625, 1.447265625, -0.69677734375, -1.0908203125, 0.460205078125, 0.51953125, -0.57373046875, 0.31298828125, 1.154296875, 0.81689453125, -1.2177734375, 0.5390625, -0.7001953125, 0.70654296875, -1.5126953125, 0.537109375, -1.3173828125, -1.2685546875, -1.58984375, -0.67236328125, -0.68408203125, -0.79638671875, 1.341796875, -0.7294921875, 0.07794189453125, -0.2841796875, 0.044708251953125, -1.029296875, 0.87353515625, -1.6123046875, 0.240966796875, 0.57861328125, -1.3369140625, -1.2578125, 0.3896484375, 3.05078125, 0.118408203125, -0.0850830078125, 0.63623046875, -0.84228515625, -0.41162109375, -0.411865234375, -1.4404296875, 0.31396484375, 0.73291015625, 0.449462890625, 0.01131439208984375, 1.1845703125, 2.150390625, 0.89013671875, 0.939453125, -0.469970703125, -1.099609375, -0.85791015625, -0.58984375, 0.4482421875, -0.074462890625, -0.28759765625, -1.3369140625, -1.7099609375, 0.376220703125, -0.86181640625, 2.048828125, -1.2197265625, 1.3916015625, -1.0390625, 1.1884765625, -1.048828125, -0.28857421875, 0.365966796875, -0.021942138671875, -1.13671875, -0.88330078125, 0.36572265625, -0.08648681640625, -0.434814453125, -0.09576416015625, -0.72216796875, -0.8388671875, -1.0107421875, -0.31298828125, -1.4794921875, 0.3955078125, -0.1507568359375, 0.162109375, 0.0092926025390625, -0.86083984375, 0.0189208984375, -0.320068359375, 0.40478515625, 0.409912109375, 0.7705078125, -1.208984375, 0.1861572265625, -0.1522216796875, 0.7724609375, 1.0078125, 2.025390625, -0.466796875, 0.4443359375, -1.2080078125, 0.04803466796875, -0.13232421875, 1.06640625, -0.406494140625, 0.041046142578125, 0.958984375, -0.14208984375, -0.72265625, 0.454345703125, -0.529296875, -0.83056640625, 1.05859375, -0.7138671875, 1.0361328125, -0.1513671875, 1.11328125, 0.75341796875, 0.69091796875, -0.4638671875, 0.99853515625, 0.04034423828125, -1.1845703125, -0.365966796875, 0.724609375, -0.007381439208984375, -0.1773681640625, 0.93359375, 1.14453125, 0.650390625, -1.25, -0.356201171875, -1.1162109375, -1.1318359375, 0.7470703125, -0.79052734375, 0.59326171875, -1.1748046875, 0.424560546875, 0.12548828125, -2.333984375, 0.116943359375, -1.8271484375, -0.0599365234375, 0.1544189453125, 0.85498046875, 0.222412109375, -1.2890625, -0.201171875, -0.2117919921875, 2.51953125, -1.353515625, 0.309326171875, 0.76708984375, -0.2110595703125, 0.39404296875, -0.181396484375, -0.35595703125, -0.310791015625, -0.67431640625, 0.7578125, 0.97607421875, 0.495849609375, 0.5390625, -0.4990234375, -0.18310546875, 1.1201171875, -0.89013671875, -1.16796875, 1.126953125, 1.501953125, -1.3125, -0.40087890625, -0.018707275390625, 0.85107421875, 0.12078857421875, 1.083984375, -0.63916015625, 0.86474609375, -2.064453125, 0.357421875, 1.138671875, 1.419921875, -1.0458984375, -1.1279296875, 1.05078125, 0.49951171875, 1.259765625, 0.056396484375, -0.91650390625, 0.85986328125, 0.52197265625, -0.251953125, -0.98095703125, 0.5107421875, -0.39599609375, -0.51416015625, 1.275390625, 1.1826171875, 1.236328125, -1.4013671875, -1.0546875, 1.8427734375, 0.419677734375, -1.9169921875, -0.06903076171875, 0.420654296875, -1.26953125, 0.79345703125, -1.7626953125, -0.370361328125, -0.59716796875, 0.59423828125, -0.7255859375, -1.060546875, 1.1865234375, -0.196044921875, 1.25390625, 0.237548828125, -1.4716796875, 0.2354736328125, 2.271484375, -0.067138671875, 0.166015625, 1.2861328125, -0.06011962890625, -0.75390625, -0.4921875, 0.84375, -0.07586669921875, 0.337158203125, -0.2222900390625, -0.73193359375, -1.0458984375, -0.252197265625, -0.6982421875, 1.2109375, -0.41943359375, -0.646484375, -0.171875, 1.1005859375, -0.051116943359375, -2.568359375, 0.94140625, 0.75927734375, -0.4267578125, -0.6845703125, -0.84375, -0.64794921875, -0.3564453125, 1.2470703125, -0.494384765625, 0.43017578125, -0.14990234375, -0.025299072265625, -0.359130859375, -1.4423828125, -1.646484375, -0.90966796875, -0.377685546875, -0.462890625, 0.7099609375, -1.5341796875, -0.701171875, -0.7880859375, 1.994140625, 1.7294921875, 1.171875, -0.71826171875, -1.1318359375, -1.4814453125, 0.50732421875, 0.60400390625, -1.30859375, -0.12353515625, -0.250732421875, 0.047821044921875, -0.20703125, 1.6982421875, 0.331298828125, 1.177734375, 0.119873046875, 0.429443359375, 0.5537109375, -1.57421875, -1.41015625, -0.60693359375, 1.173828125, -1.3525390625, 0.86181640625, -0.218017578125, -1.64453125, -0.85400390625, -2.162109375, -0.5859375, -1.8740234375, -0.5712890625, 0.185546875, -0.390625, 0.8505859375, 0.8115234375, -0.46630859375, -0.1319580078125, -2.54296875, 0.058197021484375, -0.94091796875, -0.30517578125, -1.134765625, -0.084228515625, -0.9423828125, -0.363037109375, -1.1318359375, -0.351318359375, -1.1806640625, -1.275390625, -0.327880859375, -0.6044921875, 1.1923828125, -0.1746826171875, 0.94287109375, 0.1478271484375, 1.8984375, 1.1513671875, 2.71875, 0.92431640625, -0.80126953125, 1.7294921875, 0.057708740234375, -2.16796875, -0.98583984375, 0.366455078125, 1.37109375, -1.1083984375, -1.0966796875, -0.65966796875, 2.421875, 1.2763671875, -1.748046875, -1.037109375, -1.556640625, 0.405517578125, 0.3095703125, 0.5810546875, -1.748046875, 0.295654296875, -0.71044921875, 0.444580078125, 1.6787109375, 0.52294921875, -0.2197265625, 1.0966796875, 0.2369384765625, 0.47509765625, 1.59765625, -0.299560546875, -0.175537109375, -1.068359375, 1.1123046875, 0.382080078125, -1.5576171875, 1.314453125, -1.064453125, -1.0595703125, 1.056640625, -0.37890625, -0.31396484375, -0.23681640625, -0.8564453125, 1.8642578125, 1.9794921875, 0.97900390625, -0.9208984375, -1.6923828125, -2.0625, -1.3349609375, 0.087158203125, 0.1552734375, -0.6357421875, 0.0303955078125, -0.0008592605590820312, -0.01215362548828125, 1.0185546875, 2.15625, -1.041015625, -0.258544921875, 1.4697265625, 0.56689453125, -1.3173828125, 0.459228515625, -1.0732421875, 0.25390625, -0.06927490234375, -0.2130126953125, -0.61376953125, -1.751953125, -0.006420135498046875, -0.35107421875, 0.8203125, 1.09765625, -0.5537109375, 0.07257080078125, 0.45849609375, -1.2744140625, 0.982421875, 2.048828125, -0.517578125, 1.076171875, -0.236083984375, -0.70654296875, -1.4755859375, 0.10223388671875, -0.235595703125, 1.669921875, -0.423828125, 0.0450439453125, -0.619140625, 0.2021484375, -0.432373046875, 1.2099609375, 0.49560546875, 0.50634765625, -0.43212890625, 0.767578125, -1.109375, -0.404541015625, 0.280029296875, 0.65087890625, -0.68603515625, 0.68408203125, -0.10662841796875, 0.50830078125, -0.184814453125, 1.88671875, -1.1494140625, 0.1279296875, 0.405029296875, 0.395263671875, -0.481201171875, -0.12420654296875, 1.4892578125, 0.440673828125, 0.052978515625, -0.8955078125, -0.1490478515625, 2.232421875, 0.8623046875, -0.408447265625, 1.228515625, 2.609375, -0.8193359375, 0.48291015625, -0.1829833984375, 1.2685546875, -1.7421875, 0.127197265625, 2.298828125, 0.21826171875, -0.62109375, -0.21728515625, -0.37109375, 0.455078125, -0.19384765625, -1.755859375, 0.17236328125, -0.693359375, 0.669921875, 1.3974609375, 1.05078125, -1.0244140625, -0.73046875, -0.84814453125, 0.49462890625, 0.271240234375, 0.6552734375, 0.732421875, 0.3857421875, -0.568359375, 1.15234375, -0.45068359375, -0.02789306640625, 0.728515625, -0.5791015625, -1.7353515625, 1.1142578125, 1.3291015625, 0.486572265625, -1.43359375, 0.3271484375, -0.501953125, -0.1361083984375, 1.708984375, -0.471435546875, 1.6484375, -0.336669921875, -1.1748046875, 0.703125, -2.927734375, -0.6875, 0.37744140625, -0.787109375, 0.343017578125, 0.7138671875, 0.283935546875, 0.95068359375, 0.494140625, 0.78564453125, -0.57861328125, -1.486328125, 0.26025390625, -0.156982421875, 0.0654296875, 0.264892578125, -0.29052734375, 0.1563720703125, 1.1396484375, -1.046875, -1.0810546875, 0.183837890625, 0.5263671875, 0.408447265625, -2.580078125, 0.91259765625, -0.165771484375, 1.4560546875, -1.400390625, -0.478759765625, -1.716796875, -0.76318359375, -1.6416015625, 1.22265625, -0.03521728515625, -0.3271484375, -0.0321044921875, 0.144287109375, 0.84228515625, -0.60205078125, -0.3818359375, -0.1380615234375, 1.6689453125, 0.06353759765625, 1.40625, -0.49365234375, -0.95947265625, 0.0869140625, -0.236083984375, -0.132568359375, 0.74658203125, 0.7392578125, -0.8408203125, 1.1279296875, 0.771484375, -0.50634765625, -0.60205078125, -0.00902557373046875, 0.62158203125, 0.38134765625, -0.426513671875, 1.4951171875, 0.0224456787109375, 1.7353515625, 1.677734375, -0.294189453125, 0.638671875, 0.2139892578125, -0.322998046875, 0.340576171875, 1.6640625, 0.9599609375, -1.0771484375, 0.1331787109375, 0.062744140625, 2.193359375, -0.00555419921875, 0.376708984375, 0.57666015625, 1.865234375, -0.75341796875, -1.4462890625, -0.4873046875, 0.4541015625, -1.4375, -0.097900390625, 0.9677734375, 0.00382232666015625, -1.138671875, -0.19384765625, -0.328857421875, 0.489990234375, -0.333740234375, -0.52490234375, -1.0830078125, 0.71875, 0.279296875, 0.366943359375, -1.5078125, 0.7314453125, 1.859375, 0.150390625, 0.203125, -0.058380126953125, -0.57275390625, -2.11328125, 0.533203125, 1.51953125, -0.75048828125, -0.05291748046875, 1.0126953125, -0.038604736328125, 0.51171875, -0.6591796875, -0.0236358642578125, 0.1053466796875, -1.087890625, 2.447265625, 0.38134765625, -1.2470703125, 0.2010498046875, 0.487060546875, 0.5869140625, -0.29931640625, -1.62109375, -0.337890625, -1.4677734375, 0.12066650390625, -0.085205078125, -0.73876953125, 1.099609375, 0.97705078125, 0.46728515625, -1.5888671875, 0.46044921875, 0.7666015625, -1.025390625, -1.1767578125, 0.382568359375, -0.91650390625, -0.580078125, 1.400390625, 0.0004534721374511719, -0.705078125, 1.0712890625, -1.5888671875, 1.3056640625, -0.822265625, 1.4873046875, -1.2919921875, -1.443359375, -0.8203125, -0.7451171875, -0.98974609375, 0.3037109375, -1.0849609375, 0.028167724609375, 1.005859375, 0.34912109375, 1.1123046875, -0.62744140625, -0.040191650390625, 0.052520751953125, 3.33203125, 0.445068359375, -0.6337890625, -1.2119140625, 0.8115234375, -0.59326171875, 2.232421875, -1.306640625, -1.091796875, -1.310546875, -1.296875, -0.1356201171875, 0.0633544921875, -0.3125, 1.140625, -0.0897216796875, -0.59814453125, -0.904296875, 0.385009765625, 1.64453125, -0.223876953125, 0.382568359375, 1.6337890625, -0.458740234375, -0.66748046875, -0.51171875, -0.80859375, -0.5546875, -0.7939453125, 0.701171875, 0.42138671875, -0.6796875, 0.54541015625, -0.4677734375, 1.841796875, -1.1396484375, 0.56201171875, -1.26171875, 0.0654296875, 1.4521484375, 0.53955078125, 1.8125, -0.59228515625, 1.8486328125, 0.9130859375, -0.9140625, 1.1689453125, 1.111328125, -0.1478271484375, -0.286376953125, 2.015625, -0.049224853515625, 0.87646484375, 1.6103515625, 0.279052734375, -0.64892578125, 0.47607421875, -1.73828125, -2.095703125, -0.1240234375, 1.1708984375, -0.375732421875, 0.00377655029296875, 1.2373046875, 0.60693359375, 0.134033203125, -0.609375, -1.224609375, -1.794921875, 1.505859375, 0.5986328125, 0.80859375, 1.2080078125, -0.248291015625, 2.060546875, 0.7333984375, -0.5517578125, -1.1552734375, 0.662109375, 0.1357421875, 0.376220703125, -1.365234375, 0.96826171875, -1.7783203125, -2.18359375, 1.1923828125, -0.319091796875, -0.71240234375, 0.198974609375, -0.09600830078125, -0.237548828125, -0.56640625, 0.2257080078125, -1.5068359375, -0.71484375, -0.212158203125, -0.08984375, -1.2509765625, -0.986328125, 0.79248046875, 0.9208984375, -0.325439453125, 0.625, 0.56640625, 0.455322265625, -0.4326171875, -0.53076171875, -0.83740234375, 1.3935546875, 0.069091796875, 0.72412109375, -0.27880859375, 0.76025390625, 0.53955078125, -0.84521484375, -0.473388671875, 0.80712890625, 1.2841796875, -0.2435302734375, 0.62060546875, 1.447265625, -0.026153564453125, -0.28076171875, 0.82421875, 0.2210693359375, -1.4853515625, -1.1572265625, -0.208740234375, -1.857421875, -1.7890625, -0.26171875, -2.38671875, 0.74853515625, -0.57470703125, -0.66064453125, 0.412109375, 1.166015625, -0.399169921875, 0.53662109375, 0.91064453125, 0.489013671875, -1.4599609375, -0.57470703125, -0.453125, 1.5224609375, 0.5146484375, 0.46728515625, -1.81640625, -1.109375, -0.349853515625, -0.007709503173828125, -0.6318359375, 0.4111328125, -1.064453125, 0.7333984375, 0.54248046875, 0.08349609375, -1.806640625, -0.60546875, -0.70166015625, -0.07611083984375, 1.0576171875, -0.10980224609375, -0.6787109375, 1.416015625, -0.243896484375, -1.76171875, -0.78076171875, 0.61865234375, -1.2490234375, 0.8369140625, 0.26025390625, -0.041656494140625, 2.25, -0.173583984375, 0.447021484375, 0.119140625, -1.34765625, 2.337890625, 0.58349609375, -0.04669189453125, -0.061737060546875, -1.072265625, 2.0078125, -1.6513671875, 0.71533203125, 2.173828125, 0.2269287109375, -0.7470703125, -0.405517578125, 0.89697265625, 1.0498046875, -0.62158203125, -0.59130859375, -1.580078125, -1.2890625, -0.9619140625, -0.4677734375, -1.6787109375, -1.6416015625, -0.77587890625, 1.697265625, -0.9345703125, 0.888671875, 0.79443359375, 0.448974609375, 1.439453125, -1.0283203125, 0.055908203125, 0.23388671875, 1.353515625, -2.046875, 0.134521484375, 0.06591796875, 1.427734375, -0.0760498046875, -0.89013671875, 0.68310546875, -0.36181640625, 2.16015625, 0.9345703125, 0.385986328125, 1.1640625, -1.0888671875, -0.09942626953125, 0.07623291015625, -2.4921875, -0.65771484375, 0.2127685546875, -0.1322021484375, -0.6572265625, -0.5322265625, -1.048828125, 1.208984375, -1.0869140625, 1.0712890625, -0.8037109375, -1.2314453125, -0.701171875, 0.70068359375, -0.9755859375, 1.1328125, 2.556640625, 0.3017578125, 1.533203125, -0.62158203125, 1.4453125, -0.37548828125, 0.72900390625, 0.18359375, 1.5654296875, -0.6328125, -0.151123046875, -1.982421875, -0.76953125, -0.430419921875, 0.379638671875, -0.8544921875, -0.2489013671875, -0.11444091796875, -0.042633056640625, 0.97802734375, -0.61669921875, -0.2215576171875, -0.51904296875, 1.427734375, -0.3828125, -0.469970703125, -0.0212860107421875, 0.30224609375, 1.0595703125, -0.6630859375, -0.568359375, -1.3515625, -0.344482421875, -1.673828125, 1.4208984375, -0.334716796875, 0.1123046875, -0.82080078125, 0.069580078125, 0.047027587890625, -0.70361328125, -0.260986328125, 1.15234375, -0.9638671875, -0.165771484375, 0.25927734375, 1.21875, 0.6923828125, -0.599609375, 1.9013671875, 0.85546875, -1.4267578125, -0.666015625, 1.0478515625, -0.27587890625, -1.2587890625, -0.26513671875, 0.84423828125, 1.056640625, 0.8955078125, 0.0242919921875, -2.423828125, -1.796875, -1.140625, 0.49609375, 0.6357421875, 0.48583984375, 0.40869140625, 0.23388671875, -0.045745849609375, -0.2127685546875, 1.580078125, -0.97509765625, -2.08984375, 1.28515625, 0.11419677734375, 0.60302734375, -0.071044921875, 2.22265625, 0.493408203125, 0.388671875, 0.50634765625, -0.7216796875, 0.01552581787109375, 1.5712890625, 0.68505859375, -1.0341796875, -1.9365234375, -0.248779296875, -0.1336669921875, 1.755859375, 0.471923828125, 0.0224456787109375, -0.0767822265625, -1.2197265625, 0.094970703125, 1.4482421875, 0.387939453125, -0.386474609375, 0.482177734375, 1.2158203125, 1.21875, 1.3935546875, -0.55517578125, 1.931640625, -1.119140625, 0.369140625, -0.68701171875, 2.271484375, -0.490234375, 1.025390625, -0.158447265625, 0.771484375, 2.3359375, -1.169921875, 1.9580078125, -0.55810546875, -0.1593017578125, 0.1553955078125, 1.8583984375, 1.2265625, 0.87548828125, -0.1905517578125, 0.76513671875, -1.544921875, 0.414306640625, 0.78955078125, -0.1781005859375, -0.7109375, -0.28173828125, 0.5205078125, -0.0682373046875, 0.2276611328125, 0.54296875, -0.0160980224609375, -0.58837890625, 0.71826171875, -0.229736328125, -1.212890625, -0.38623046875, 0.260986328125, 0.468994140625, 0.09197998046875, 0.34228515625, -1.2998046875, -1.6357421875, 0.4423828125, -1.0732421875, -1.1083984375, -0.298828125, -1.669921875, 0.81640625, 1.150390625, 1.1962890625, 0.4365234375, 0.00466156005859375, -0.97216796875, -2.0390625, -0.64599609375, 0.440673828125, -0.619140625, -0.95361328125, -0.80322265625, 1.0947265625, -1.0322265625, -1.6220703125, -0.8330078125, -0.19921875, -1.1650390625, -0.416015625, 1.748046875, 1.482421875, -0.505859375, 0.75244140625, 0.85986328125, -0.26708984375, -0.79638671875, 0.443359375, 1.0517578125, 0.97119140625, 1.80859375, -0.93701171875, -0.9423828125, -0.87255859375, 1.404296875, 0.8994140625, -1.3837890625, 0.4501953125, -1.2265625, 0.57177734375, 0.64990234375, 1.580078125, -0.89794921875, -1.2333984375, 0.998046875, 0.9033203125, 0.146728515625, 0.00518035888671875, -1.34375, 1.0419921875, 2.095703125, -0.92041015625, 0.513671875, 0.90966796875, -0.3671875, -0.062164306640625, 1.126953125, -1.0107421875, -0.10260009765625, -0.9375, 0.6728515625, 0.5556640625, 1.7119140625, 0.289794921875, 0.80419921875, -1.884765625, 0.505859375, -0.9853515625, 1.9443359375, 0.040740966796875, -1.91796875, -0.6826171875, -0.005977630615234375, -0.0037174224853515625, -0.006771087646484375, -0.399169921875, -0.0246429443359375, 1.2919921875, -0.67578125, -1.1513671875, -1.599609375, -0.66015625, 0.372314453125, -1.9345703125, -0.138916015625, -0.15234375, 1.056640625, 0.720703125, 0.391357421875, -1.9306640625, 0.501953125, 0.65185546875, -1.1962890625, 0.30126953125, 0.00768280029296875, -0.2154541015625, 0.1722412109375, -0.03765869140625, -0.6708984375, -0.222900390625, -0.499755859375, -0.0323486328125, 1.1953125, -0.469970703125, 0.8544921875, 0.54833984375, -0.70654296875, -0.68896484375, -1.447265625, -0.271484375, 0.445068359375, -1.123046875, -0.411376953125, -0.2105712890625, 2.634765625, 0.290771484375, -0.317138671875, -1.0849609375, 1.0302734375, 0.60205078125, -0.343994140625, -1.87109375, -0.4140625, -1.087890625, 0.96435546875, 0.05364990234375, -0.79443359375, 0.685546875, -1.42578125, -0.297119140625, -0.409912109375, -0.71630859375, 0.2037353515625, -0.47412109375, 1.3349609375, -1.2158203125, 0.62158203125, 0.939453125, -0.441162109375, -0.49609375, -1.02734375, 1.3154296875, 0.6416015625, 0.3896484375, -1.09765625, 2.390625, 0.50537109375, 1.279296875, 0.93212890625, 0.8818359375, -2.13671875, -0.66259765625, 0.2264404296875, -1.2236328125, 0.4228515625, 0.453125, -3.0078125, -1.7119140625, -0.2213134765625, 0.2049560546875, 0.3759765625, 1.560546875, 1.98046875, 0.387939453125, 0.9306640625, 0.1656494140625, 0.02984619140625, 0.1395263671875, 0.970703125, -0.366455078125, 0.13427734375, 1.5361328125, -0.10675048828125, 1.169921875, -2.638671875, 1.396484375, 1.138671875, 0.595703125, -0.2265625, 1.0419921875, -0.78125, 0.54833984375, 0.44482421875, 0.250732421875, -0.2607421875, -0.16845703125, 0.1754150390625, -0.896484375, 1.51953125, 0.546875, 0.87255859375, 0.58837890625, -0.12286376953125, -2.248046875, -0.71728515625, 0.8662109375, 0.7080078125, 0.826171875, -0.63134765625, 0.266357421875, -0.33251953125, -0.27783203125, 0.75830078125, -0.2286376953125, 0.95166015625, -1.2607421875, -0.348876953125, -0.344482421875, -0.16455078125, 0.556640625, -2.033203125, 1.0244140625, -0.572265625, 0.176513671875, 0.9541015625, -0.30126953125, -1.552734375, 1.064453125, 1.2041015625, -0.491943359375, 1.3955078125, 2.994140625, 0.63037109375, 0.126708984375, -1.02734375, -0.78662109375, 1.2392578125, -0.36279296875, 0.8193359375, -1.857421875, -0.1533203125, 1.232421875, -0.99609375, 0.55615234375, 2.251953125, 1.0771484375, 0.7158203125, 1.94140625, 0.36328125, 0.23486328125, -0.1483154296875, 0.369140625, -0.0457763671875, -0.67578125, 0.042266845703125, -0.67578125, 0.331787109375, -0.078369140625, -0.87255859375, 1.5166015625, 0.7060546875, -0.9033203125, 0.939453125, -0.0230560302734375, 0.054718017578125, 0.904296875, 0.3759765625, -0.912109375, 1.7919921875, -1.0947265625, -0.390380859375, -1.06640625, 1.59765625, -0.2646484375, -1.8291015625, 1.6826171875, 0.370849609375, 0.403076171875, 0.2265625, -0.6826171875, -0.212158203125, 1.642578125, -0.248779296875, 1.2080078125, -0.1422119140625, -1.30078125, -0.244140625, -1.1416015625, -0.9326171875, -0.9169921875, -1.72265625, 0.385009765625, 0.364013671875, -0.261474609375, 0.6435546875, 0.08184814453125, -1.0341796875, 0.5751953125, -0.1865234375, 0.232666015625, -0.324462890625, -0.314453125, -1.2861328125, 0.5751953125, -0.06451416015625, 0.5537109375, 0.297607421875, 0.31591796875, 0.783203125, 0.227783203125, -0.2490234375, -0.2213134765625, -0.357421875, -1.13671875, 0.422119140625, 0.057861328125, 0.7568359375, 0.201416015625, 1.5537109375, -1.43359375, -0.2158203125, -1.3017578125, -1.38671875, 1.919921875, 0.0106048583984375, 1.2451171875, -1.4755859375, -1.146484375, 0.5234375, 2.17578125, 0.81640625, 0.31005859375, -0.2193603515625, -2.443359375, -0.58203125, -0.77001953125, 1.0986328125, -1.84765625, 0.8203125, -0.4375, -0.26513671875, -1.73828125, -0.1451416015625, -0.084716796875, 0.289794921875, -1.720703125, 1.4482421875, 0.517578125, 0.94287109375, 0.6416015625, -0.748046875, 1.21875, 0.424072265625, -1.1103515625, -0.9169921875, -0.174072265625, -0.464111328125, -1.818359375, 0.2159423828125, 1.5517578125, 0.1533203125, -2.20703125, -1.1123046875, -0.27197265625, -1.056640625, 0.2203369140625, -0.84326171875, -0.54296875, 0.3154296875, -1.4306640625, -1.9306640625, 0.41162109375, 0.0164337158203125, 1.4248046875, 0.8037109375, 0.95947265625, -0.0428466796875, -1.091796875, -0.787109375, 1.87890625, -0.315673828125, 0.474853515625, -0.1336669921875, -0.9794921875, 0.29541015625, 1.513671875, 0.494873046875, -0.341796875, -0.98974609375, 0.5185546875, 1.0146484375, -2.0546875, 0.11688232421875, -0.53369140625, 0.537109375, -0.83544921875, 0.326416015625, -0.30712890625, 1.4521484375, -1.3095703125, 0.1104736328125, -0.79248046875, -1.421875, 0.89013671875, 0.420654296875, -0.141357421875, -0.409423828125, 0.4072265625, -0.020599365234375, -1.4130859375, -1.53125, 0.6435546875, -0.5556640625, 0.81689453125, 0.4921875, -1.6875, 0.152587890625, -0.83447265625, 0.428466796875, 0.056884765625, 1.294921875, -0.2008056640625, -0.82080078125, -0.65185546875, 0.65576171875, 1.4912109375, 1.3046875, 0.461669921875, -0.27880859375, -0.95703125, -0.66552734375, 2.1015625, -0.23828125, -0.38623046875, 2.294921875, -1.720703125, -1.8408203125, 0.28369140625, 0.239013671875, -0.45263671875, -0.048065185546875, 2.052734375, -0.7958984375, 2.591796875, -0.10443115234375, -1.1845703125, -0.34912109375, -0.50341796875, -0.031036376953125, -1.310546875, -0.1964111328125, 0.896484375, 1.00390625, 0.78466796875, 2.6875, 0.29736328125, -0.4716796875, 1.49609375, 1.060546875, 0.6240234375, 0.1485595703125, -0.771484375, -0.4013671875, -1.1162109375, 0.2235107421875, -1.5859375, -0.84521484375, -1.083984375, 0.75634765625, -0.01381683349609375, -0.54541015625, -0.82080078125, -2.11328125, -1.189453125, -1.8349609375, 1.455078125, -0.375732421875, -0.166748046875, -0.154296875, -0.25048828125, 0.810546875, -0.3369140625, -0.73486328125, -1.1318359375, 0.38427734375, -0.1156005859375, 1.38671875, 0.6337890625, -1.8193359375, 1.3935546875, -0.273193359375, 0.60107421875, -0.078857421875, -1.154296875, 0.0207977294921875, -2.140625, -0.9111328125, 1.8232421875, -0.8818359375, -0.5517578125, 0.020050048828125, -1.544921875, 1.0732421875, 1.3720703125, 0.358154296875, -1.4013671875, -0.030487060546875, -0.09246826171875, -0.72509765625, 1.1689453125, -0.1419677734375, -1.3154296875, -0.32275390625, 0.95166015625, 1.025390625, -0.6640625, -0.89697265625, -0.0189361572265625, 0.0303192138671875, -0.2476806640625, 0.763671875, 0.48779296875, 2.0390625, -0.37548828125, -0.93408203125, 0.62744140625, 0.11578369140625, 0.54833984375, 0.04522705078125, 0.037872314453125, -0.724609375, 1.7783203125, -1.3388671875, -0.130859375, -0.482666015625, -1.0908203125, -1.625, -0.87548828125, -2.32421875, 1.265625, -0.48681640625, 1.0673828125, -0.53466796875, -1.130859375, -1.2509765625, 0.302490234375, -1.2529296875, -1.923828125, 0.83056640625, -0.96044921875, -0.02081298828125, -1.31640625, -0.253662109375, -0.219482421875, 2.02734375, -0.98779296875, 0.623046875, -0.65576171875, -0.53076171875, 2.505859375, 1.294921875, -0.78125, -0.30908203125, -0.45849609375, 2.47265625, -0.046875, -0.33837890625, 0.7998046875, -0.763671875, -0.462890625, -1.21875, 0.8662109375, -0.058807373046875, 1.6279296875, 1.8623046875, -1.0263671875, -1.5498046875, 1.2978515625, 0.9169921875, -0.1885986328125, -0.50927734375, -0.048736572265625, 0.3896484375, -1.021484375, 0.045623779296875, 1.2099609375, -0.2098388671875, -1.4580078125, 1.56640625, -1.607421875, -0.469970703125, 1.3828125, 0.3662109375, -0.260498046875, -2.861328125, -0.92236328125, -0.7763671875, -0.455810546875, 0.33349609375, 0.1373291015625, 0.2396240234375, -1.841796875, -0.032958984375, 0.300048828125, 0.1788330078125, -1.11328125, 1.2353515625, 1.0087890625, -0.47314453125, 0.03692626953125, -0.1785888671875, -2.498046875, -0.466552734375, -1.1796875, 0.0234527587890625, -0.6630859375, 2.5234375, 1.4609375, -0.151611328125, -0.677734375, 0.62744140625, 1.4296875, 1.953125, -0.84375, -0.68212890625, -1.462890625, 0.07061767578125, -0.239990234375, 1.068359375, 0.55029296875, 0.66357421875, -1.0322265625, -2.58203125, -1.6083984375, -0.88525390625, 0.69921875, -2.158203125, -0.35595703125, -1.1875, 0.7275390625, 0.91943359375, 1.1982421875, 1.53515625, -0.66162109375, -1.7275390625, 0.15576171875, -0.59130859375, 0.6728515625, -0.49755859375, 0.640625, 0.85986328125, 0.44482421875, 0.184326171875, -1.2705078125, 0.7861328125, -0.7705078125, -0.396728515625, -0.4794921875, 0.662109375, 1.30078125, 0.52294921875, 0.89013671875, -1.4287109375, -1.0341796875, 0.57177734375, -0.69384765625, -0.75634765625, 0.260986328125, 0.69091796875, 0.451904296875, 0.253662109375, 0.005764007568359375, -0.1907958984375, -0.712890625, 1.1748046875, -1.34375, -0.055938720703125, -0.04779052734375, -0.5439453125, -0.0966796875, -0.78515625, -0.8759765625, 0.0333251953125, 0.20654296875, 0.002651214599609375, 0.48828125, -0.9794921875, -1.5986328125, 0.498291015625, -2.228515625, 0.646484375, -0.76171875, -1.2431640625, -0.29541015625, -0.623046875, 0.87890625, 0.07373046875, -0.8037109375, -0.69384765625, 1.044921875, -1.796875, -0.72314453125, -0.60205078125, 2.109375, 1.0400390625, 1.212890625, -1.0712890625, -1.7880859375, -1.7197265625, -0.1519775390625, -0.994140625, 0.892578125, 1.8916015625, 0.27197265625, 0.474853515625, 0.43701171875, 0.1796875, -0.10858154296875, 0.336181640625, 0.12335205078125, -1.7607421875, -1.5810546875, -0.9267578125, 2.056640625, -0.900390625, 0.66455078125, -1.1669921875, 0.257568359375, -0.54541015625, -1.3203125, -1.1064453125, -0.134033203125, -0.0933837890625, 1.6630859375, 1.8154296875, 0.6416015625, 0.406005859375, 0.95556640625, -2.447265625, -1.7041015625, -0.2646484375, 0.970703125, 1.1376953125, -2.00390625, -1.1787109375, -1.001953125, 0.10504150390625, -1.8955078125, -0.41552734375, 0.03662109375, -0.60107421875, -0.0855712890625, 0.87353515625, 0.33203125, 0.270263671875, -2.31640625, 1.0087890625, -1.2490234375, -0.236328125, -0.158935546875, -1.509765625, 0.25634765625, -0.83740234375, 1.1455078125, -0.277099609375, 0.9921875, 0.14599609375, 0.1695556640625, -2.099609375, -0.56982421875, -0.274658203125, 1.390625, 1.03125, 0.5908203125, 0.40185546875, -0.07415771484375, 0.36962890625, -1.318359375, -1.94140625, 0.3994140625, 2.1796875, 2.162109375, -0.470703125, -0.266845703125, -0.576171875, -0.30615234375, 0.4619140625, -1.32421875, 0.607421875, -0.32568359375, -0.52392578125, -2.13671875, 0.1983642578125, 0.4658203125, -0.32958984375, -0.11248779296875, 1.654296875, -2.17578125, 0.56640625, -0.85302734375, 1.1845703125, -0.83544921875, 0.049407958984375, -0.428955078125, 0.4951171875, -1.7939453125, 1.4853515625, 1.2841796875, 0.046173095703125, -0.64501953125, 1.345703125, -0.55615234375, 0.54541015625, 0.320068359375, -0.298828125, -0.8330078125, 0.75146484375, 1.6591796875, -0.26416015625, 1.517578125, 0.73046875, -0.759765625, 0.5693359375, -0.52978515625, 0.74462890625, -1.5625, 0.15576171875, 1.6787109375, -0.320556640625, 1.47265625, 0.1810302734375, -1.7958984375, 1.232421875, 1.0380859375, -1.2978515625, 1.654296875, 0.6181640625, 0.1810302734375, -0.6669921875, 1.037109375, -0.83154296875, -0.82421875, 1.7734375, -1.5546875, 0.9150390625, 0.8486328125, 1.17578125, -0.290771484375, -0.89697265625, 0.1962890625, -0.61474609375, 0.89990234375, 1.28125, -0.1300048828125, -0.3740234375, -0.59521484375, 2.123046875, -0.297607421875, -0.73974609375, -1.783203125, 1.7490234375, -0.84716796875, -0.60986328125, 0.1812744140625, 0.984375, -1.4140625, -1.0380859375, 0.0870361328125, -0.78369140625, 1.5595703125, 2.431640625, 2.056640625, 1.3623046875, 1.3173828125, 2.841796875, -1.314453125, 0.3818359375, -0.81396484375, 0.0291290283203125, 2.443359375, 0.8076171875, -1.0361328125, 0.1531982421875, -2.248046875, -0.070068359375, -1.7138671875, 1.5341796875, 0.97802734375, 1.1025390625, 0.90478515625, 0.947265625, 1.349609375, -1.56640625, 1.80078125, 0.85302734375, 0.47412109375, -0.53271484375, 1.9794921875, -0.8115234375, -0.1871337890625, -0.1500244140625, 0.291748046875, -1.060546875, 0.7353515625, 0.211669921875, -1.1591796875, 1.4267578125, -1.1572265625, -1.087890625, -1.1962890625, 0.9384765625, -0.494140625, 0.6708984375, -1.546875, 1.4296875, 0.87060546875, -0.05474853515625, -0.395263671875, -0.53662109375, -0.6455078125, -0.40771484375, -1.7685546875, 1.853515625, 1.044921875, -0.293701171875, 0.8544921875, -1.4326171875, -0.0672607421875, 1.0458984375, 0.89697265625, 0.5048828125, 0.0292816162109375, -1.1162109375, 1.1650390625, 0.2091064453125, -1.1455078125, -1.1865234375, 0.55126953125, -1.2412109375, -1.6513671875, -0.255615234375, 1.9111328125, -1.1435546875, 0.98828125, -1.392578125, -1.2265625, -0.8837890625, -1.3916015625, 1.44140625, -0.69873046875, 0.1656494140625, -0.380615234375, 0.377197265625, 0.30029296875, -0.7568359375, 1.533203125, 0.8623046875, 0.6318359375, -0.2406005859375, 0.09173583984375, -0.053680419921875, -1.94140625, -1.60546875, 0.71142578125, 0.290283203125, 0.45751953125, -0.8193359375, -0.935546875, -1.208984375, 0.80029296875, 0.88525390625, 1.20703125, -0.0276031494140625, -0.54150390625, -0.78955078125, -0.3427734375, -0.2215576171875, 1.451171875, 0.1949462890625, 0.38037109375, -0.7578125, 0.2373046875, 0.5732421875, -0.07220458984375, -1.12109375, 1.6513671875, -0.039825439453125, 1.06640625, 0.07568359375, 0.607421875, 0.65380859375, -0.79833984375, 0.263916015625, -0.42431640625, 0.2724609375, -0.73193359375, -0.76416015625, 0.79931640625, -1.1259765625, 0.8896484375, 0.291748046875, -0.98583984375, 0.5693359375, -0.98388671875, -0.62548828125, 0.42431640625, 0.2264404296875, 0.1029052734375, 0.2083740234375, 0.6650390625, -0.07708740234375, 0.1336669921875, 2.162109375, -1.51953125, 0.5576171875, 0.263671875, 0.304443359375, 0.487548828125, 0.403076171875, -0.1822509765625, 1.45703125, 0.331787109375, -0.10260009765625, 2.779296875, -0.09051513671875, 0.84619140625, -0.34423828125, 0.86083984375, -0.19921875, -1.2578125, -1.0009765625, -1.578125, -0.67138671875, 0.03143310546875, -0.60546875, 0.203857421875, 1.7109375, 1.646484375, -0.0701904296875, 2.720703125, 0.8740234375, 0.84423828125, -0.58740234375, 0.253662109375, -0.38525390625, 0.0274505615234375, -0.99560546875, 0.434326171875, -0.83984375, -0.09112548828125, -0.404541015625, 1.4521484375, 0.51708984375, -0.5478515625, -0.56396484375, -0.307373046875, 0.40380859375, -0.64111328125, -2.142578125, 0.436279296875, -1.404296875, 0.46337890625, -0.59912109375, 2.234375, -0.1279296875, 0.73193359375, -0.61767578125, -0.876953125, 1.4296875, 0.1318359375, 0.90966796875, -0.6064453125, 0.244873046875, -0.625, 0.337890625, -0.485595703125, 0.970703125, -0.1380615234375, -0.298583984375, -0.611328125, -0.342529296875, -1.357421875, 0.04388427734375, 0.325439453125, -2.12109375, 0.342529296875, 0.10595703125, 1.4638671875, -0.450927734375, 0.167724609375, -1.0234375, 3.369140625, 0.146484375, -0.556640625, 1.8330078125, -0.58154296875, 0.05950927734375, -0.845703125, 1.1962890625, -0.15625, 0.45361328125, 0.61669921875, 1.2216796875, 1.4541015625, 0.51953125, -0.6142578125, -0.5810546875, 0.81396484375, 0.133544921875, 0.640625, -0.1151123046875, 0.44873046875, 0.361328125, -0.68701171875, 1.625, 0.427734375, 1.7294921875, 0.2469482421875, 0.7021484375, -0.2069091796875, 0.87158203125, 0.2685546875, 0.07086181640625, 1.607421875, -0.73974609375, 2.306640625, 0.92041015625, -0.187255859375, 0.6474609375, 1.443359375, 0.245849609375, 0.1590576171875, 0.3232421875, 1.486328125, 0.83740234375, -0.137451171875, -0.607421875, -2.21484375, 1.29296875, -1.154296875, 0.2474365234375, -1.244140625, -1.220703125, 0.08331298828125, -0.1358642578125, 0.8359375, -1.66015625, -1.25390625, -0.1807861328125, 0.58642578125, 0.02618408203125, 1.1015625, 0.405029296875, -0.1370849609375, 1.20703125, -0.69970703125, -0.64599609375, 0.1246337890625, 0.896484375, 0.7060546875, 0.205322265625, -0.50244140625, 0.06988525390625, -1.6259765625, -0.316162109375, 0.173583984375, 0.93896484375, -0.5849609375, 0.302978515625, -1.8662109375, -1.2294921875, -1.4013671875, -0.1405029296875, 1.115234375, -1.115234375, -1.326171875, 0.2015380859375, -1.2646484375, 1.263671875, 0.64990234375, -0.859375, 0.312255859375, -0.116455078125, -0.3310546875, 0.8330078125, -1.203125, -0.8916015625, 0.49560546875, -0.455810546875, -1.255859375, -1.4091796875, 0.36865234375, -1.109375, 0.0843505859375, 0.385009765625, 1.0224609375, -0.483154296875, -0.282470703125, -1.005859375, 0.95556640625, 0.163818359375, -0.5029296875, -0.0496826171875, 0.283203125, 0.70751953125, -1.166015625, -0.0020847320556640625, -0.78515625, -1.9931640625, -0.52001953125, -0.434814453125, -2.1171875, -0.11444091796875, -0.92236328125, 0.95703125, 1.4267578125, -1.65234375, 0.89404296875, -1.4541015625, 0.089111328125, -0.1019287109375, -0.8505859375, 0.63525390625, -0.8818359375, 0.50634765625, 0.275390625, -1.91015625, -0.29345703125, 0.004703521728515625, -0.5830078125, -0.0838623046875, -0.2252197265625, -0.492431640625, 1.2900390625, -0.1861572265625, -0.1314697265625, -1.244140625, 0.9755859375, -0.1270751953125, -0.1912841796875, -0.321044921875, 0.0253448486328125, 0.251708984375, 0.62060546875, -0.63037109375, 2.220703125, -1.515625, 0.89794921875, 0.427001953125, -0.09820556640625, 0.142578125, 0.21337890625, -0.841796875, 1.9580078125, 0.7626953125, -0.74072265625, -1.2587890625, 0.771484375, 1.2158203125, 0.302734375, 1.095703125, -0.0145721435546875, -0.127197265625, -0.81982421875, 0.1258544921875, 0.351318359375, -0.1097412109375, -0.724609375, 0.38232421875, -0.650390625, -0.2135009765625, 1.2001953125, -1.0576171875, 0.301513671875, 0.51611328125, -0.365966796875, -0.2213134765625, 0.329833984375, -1.013671875, 1.4443359375, -1.13671875, 0.27783203125, -0.19580078125, 0.57177734375, 0.255126953125, 1.796875, -0.82373046875, -0.1708984375, -0.0999755859375, -2.76171875, 0.63916015625, 1.099609375, -0.250244140625, 0.0180511474609375, 0.63134765625, -0.6591796875, 1.703125, 1.9306640625, -0.6259765625, 2.306640625, 0.33935546875, -0.9609375, 0.046722412109375, 0.051605224609375, -0.89990234375, 0.1978759765625, 1.8154296875, 0.96826171875, -0.40625, 0.47021484375, -0.395751953125, -0.1719970703125, -0.168212890625, 0.8544921875, 0.255859375, -0.306640625, 0.404296875, 0.58544921875, 1.4833984375, 0.58740234375, 0.66162109375, -0.00936126708984375, -0.56396484375, -1.4169921875, 0.7080078125, -0.447265625, -0.54931640625, 0.179443359375, 1.3828125, 0.20263671875, 0.055084228515625, 0.100830078125, -0.326171875, -1.265625, 3.01171875, -2.1640625, 2.76953125, 1.9638671875, -0.339599609375, 0.059661865234375, 1.42578125, -0.81298828125, 0.55322265625, -0.1826171875, -0.1260986328125, -1.1259765625, 0.53173828125, 0.280517578125, -2.28125, 0.0012197494506835938, -0.32373046875, -1.251953125, 1.5556640625, -1.21875, 0.07666015625, 0.3642578125, 0.57568359375, -0.9111328125, 0.08734130859375, -0.034149169921875, -0.837890625, -1.07421875, -0.54541015625, -0.94384765625, -0.93603515625, 0.2113037109375, -1.7734375, 1.2685546875, 0.1405029296875, -1.8828125, -0.84716796875, -0.3203125, -0.040924072265625, 0.89111328125, -0.95703125, -0.85302734375, -0.798828125, 0.0728759765625, 0.327880859375, -0.10418701171875, 0.1402587890625, -2.333984375, -0.298828125, 0.356201171875, -0.482177734375, 1.1728515625, -1.021484375, -0.92236328125, 1.1298828125, -0.1732177734375, -1.09765625, -0.1663818359375, 0.2357177734375, -1.0009765625, 1.6298828125, 0.327880859375, -0.006877899169921875, 1.16796875, 0.61279296875, -0.71435546875, 0.20556640625, 0.2626953125, 1.16796875, 2.060546875, 0.5078125, 0.9287109375, 0.456298828125, 2.15625, 0.242431640625, 0.599609375, -0.475830078125, 0.037872314453125, -0.8515625, 0.347412109375, 0.325439453125, -1.126953125, -0.8837890625, -0.72265625, 1.38671875, 0.00945281982421875, 1.3017578125, 1.017578125, -0.984375, 0.195068359375, 0.379638671875, -1.0654296875, -0.2724609375, -1.380859375, 1.1328125, 0.444580078125, -0.56884765625, 0.2144775390625, -1.1806640625, 0.1572265625, 0.3369140625, -0.9296875, -2.0625, 0.443115234375, 1.212890625, 0.495361328125, 1.021484375, -0.046905517578125, 1.67578125, 0.6806640625, -0.6875, 1.673828125, -1.89453125, 0.0872802734375, -0.32861328125, -0.11358642578125, 0.53076171875, 1.8818359375, 0.96044921875, -0.10540771484375, -1.416015625, 1.03125, -0.6630859375, -0.1278076171875, -0.53466796875, -0.60009765625, 0.0211944580078125, 1.2919921875, -1.3154296875, 1.0419921875, 0.0816650390625, -2.080078125, -0.274169921875, -0.65234375, -0.5009765625, -0.38037109375, 0.86962890625, -0.3076171875, 1.119140625, 1.177734375, -0.7373046875, 1.1748046875, 0.418701171875, 1.3466796875, 0.1566162109375, 0.496337890625, -0.092529296875, 0.04107666015625, -0.93505859375, 0.7568359375, -0.68701171875, 2.37890625, 0.309326171875, -0.2305908203125, -1.021484375, 0.10650634765625, -0.80224609375, -1.4345703125, 0.291259765625, -0.52001953125, -2.37109375, 0.042816162109375, -0.42919921875, -0.5625, 0.26513671875, 0.5947265625, 2.171875, -1.0791015625, -1.1923828125, -2.509765625, 1.4599609375, -0.259765625, -1.0849609375, 0.2264404296875, 0.919921875, -0.62060546875, -0.8740234375, -0.70068359375, 1.4833984375, -0.69287109375, 0.38720703125, 1.7646484375, 0.201171875, 0.970703125, 0.8662109375, -0.1337890625, -0.369873046875, -0.40625, 0.1475830078125, 1.013671875, -1.2421875, 0.005329132080078125, 0.923828125, -0.1341552734375, 0.71875, -0.2412109375, -0.50634765625, 1.423828125, 0.333984375, 0.98779296875, 0.98046875, -1.6240234375, -0.22314453125, 0.89208984375, 0.12188720703125, 0.54296875, -0.437255859375, -0.281005859375, -0.457763671875, -0.31103515625, -0.348876953125, -0.52197265625, -0.489501953125, -0.326171875, 1.1005859375, 1.3671875, -0.6640625, 0.03070068359375, 0.049957275390625, -0.82177734375, -0.1650390625, -2.716796875, 0.1884765625, -0.7421875, 0.8935546875, 0.497314453125, 0.06298828125, 1.611328125, 0.142333984375, -1.587890625, -0.513671875, 0.399169921875, 1.296875, -0.77294921875, 0.393310546875, 0.61767578125, -0.2626953125, -0.07867431640625, 0.56201171875, 1.08203125, 0.6416015625, -0.385986328125, 0.1376953125, -0.09405517578125, 0.042449951171875, 0.03863525390625, 1.0283203125, 0.1751708984375, -0.7265625, -0.70751953125, 2.439453125, 0.732421875, -0.60205078125, -1.056640625, -0.232177734375, 0.04998779296875, 0.1756591796875, -0.9208984375, 0.60546875, 0.3876953125, 0.251708984375, 0.39892578125, 0.9130859375, -1.0263671875, 0.1285400390625, 1.2333984375, 1.515625, 1.63671875, -0.28662109375, 0.045867919921875, -0.529296875, -1.1142578125, 0.20458984375, -0.07550048828125, -0.9072265625, -1.17578125, -1.810546875, 2.451171875, -0.6845703125, -0.3798828125, -0.4521484375, -0.51025390625, -0.2169189453125, -0.72021484375, 0.1878662109375, -1.7841796875, 1.0341796875, -0.08770751953125, -0.278564453125, -0.18798828125, 0.362548828125, -0.422607421875, 0.1605224609375, -1.42578125, 1.86328125, 0.90869140625, -0.66064453125, 0.47119140625, -1.16796875, 0.78857421875, 1.072265625, -1.486328125, -0.7607421875, 0.439208984375, -1.0126953125, 0.67578125, 1.294921875, 0.5791015625, 0.3447265625, 1.0498046875, -1.4404296875, 1.837890625, 0.333984375, 0.2108154296875, -0.7236328125, -1.17578125, -1.435546875, -0.1390380859375, 0.288818359375, 2.125, 1.1767578125, 0.96923828125, 0.29150390625, 0.01922607421875, 0.403564453125, -0.2037353515625, -1.8154296875, 0.411376953125, 0.421875, -1.001953125, -1.0302734375, -0.105712890625, 0.30126953125, -0.9560546875, 0.55126953125, 1.35546875, 0.1907958984375, -0.62841796875, 2.70703125, -0.9423828125, -1.453125, 0.52001953125, -0.69970703125, -1.6162109375, -0.603515625, -0.31884765625, -2.044921875, 0.79052734375, -0.8212890625, 1.474609375, -2.0078125, -1.041015625, -0.34912109375, -0.0689697265625, -0.464111328125, -1.875, -1.279296875, -1.8408203125, 0.07415771484375, -2.4296875, 0.2242431640625, -0.2353515625, -0.89892578125, 0.669921875, -0.36962890625, -0.21630859375, -0.07568359375, 0.55810546875, -1.185546875, -0.348876953125, -0.332763671875, 0.320068359375, 0.44384765625, -0.5205078125, 0.33447265625, -0.7607421875, 0.2200927734375, -0.022003173828125, -0.406494140625, 0.69775390625, 0.8251953125, -0.39990234375, -1.69140625, -1.3818359375, 2.087890625, 0.019561767578125, 0.27734375, -0.267333984375, 0.148681640625, 1.15234375, -1.310546875, -0.732421875, -0.384033203125, -1.3994140625, 0.35107421875, -2.265625, 0.11761474609375, 0.7861328125, 0.572265625, -1.1259765625, -0.55322265625, 1.701171875, -1.908203125, -1.0087890625, 1.1103515625, 0.213134765625, 0.10150146484375, 1.2060546875, 1.01953125, -2.10546875, 0.498779296875, -1.9404296875, 1.048828125, 0.27001953125, 0.32958984375, -0.34130859375, -4.27734375, -0.464111328125, -0.07220458984375, -0.7626953125, 0.96435546875, 0.0238800048828125, 1.232421875, 0.486083984375, -0.85986328125, -0.409912109375, 0.286865234375, -0.5087890625, 0.317138671875, -0.86083984375, -0.47802734375, 0.1356201171875, -1.1611328125, -0.2083740234375, -0.1568603515625, -1.9189453125, -1.119140625, -0.40625, -0.77197265625, -0.419921875, -0.406982421875, -1.45703125, -0.71826171875, -0.248046875, 0.322265625, 0.8359375, -0.93115234375, -0.87353515625, -0.109130859375, 0.79150390625, -1.1875, -1.451171875, 2.205078125, 0.63427734375, -0.59765625, -0.30615234375, -0.189453125, -2.001953125, 0.24609375, 3.455078125, 0.026611328125, -0.62060546875, -0.72216796875, -0.76025390625, 0.06878662109375, 1.955078125, -0.4296875, -0.8359375, 1.01171875, -1.470703125, 0.145263671875, 1.2333984375, 0.487060546875, -1.6083984375, -0.33837890625, -0.77783203125, -2.177734375, -0.083740234375, 0.12200927734375, 1.7958984375, -0.63720703125, -0.0867919921875, -2.392578125, -1.048828125, 0.74365234375, -0.26220703125, -0.642578125, -1.77734375, 1.08203125, 0.22216796875, 2.10546875, -0.060394287109375, 0.70458984375, -0.334228515625, 0.003841400146484375, -0.9599609375, 0.2666015625, 0.0235137939453125, 0.83642578125, 1.30859375, -1.546875, -0.7138671875, 1.951171875, 0.6328125, -0.69287109375, -0.515625, 1.3359375, 1.8115234375, 0.7333984375, 0.681640625, -1.4013671875, 0.87451171875, -0.338134765625, -0.01934814453125, -0.203857421875, 1.19921875, 0.250244140625, 1.23828125, -1.21875, -0.84814453125, -0.61767578125, 0.07666015625, -0.183837890625, 0.65673828125, -0.99365234375, -0.642578125, -0.73486328125, -0.556640625, -0.7099609375, 0.8935546875, 0.2474365234375, -0.430419921875, -0.724609375, -2.37109375, 0.46337890625, 0.68701171875, 0.7265625, -0.366943359375, -0.429931640625, 1.2509765625, -0.806640625, -1.1484375, 0.259033203125, -0.92041015625, -0.90625, -0.51220703125, -1.478515625, 1.1669921875, 1.8583984375, -1.3857421875, -0.06756591796875, 2.130859375, 0.61962890625, -0.63232421875, 1.2294921875, -1.8369140625, 1.4609375, -0.406494140625, -0.65966796875, 2.8359375, 0.366455078125, 0.421142578125, -1.015625, -0.215576171875, -0.467529296875, 1.85546875, -1.5068359375, -0.0027675628662109375, 0.73046875, 1.94921875, 1.17578125, 1.291015625, 1.0390625, 0.568359375, 2.125, 0.336181640625, 1.3974609375, -1.4580078125, 0.2406005859375, -1.2763671875, -0.556640625, 0.455078125, 0.305419921875, -0.435302734375, -2.107421875, -0.66259765625, -1.2939453125, 0.74853515625, -1.5244140625, -1.6474609375, 0.7958984375, -1.3515625, -1.65625, 0.351318359375, 1.1875, 0.333251953125, -1.673828125, 0.370849609375, -1.4501953125, -0.892578125, 1.3623046875, -1.09765625, 0.7451171875, 2.283203125, 0.96826171875, 0.2073974609375, 0.161376953125, 0.6142578125, 0.40771484375, 1.525390625, -1.05859375, 0.59619140625, 0.70458984375, -1.4990234375, 0.298095703125, 0.2169189453125, -1.21875, -0.58642578125, -2.82421875, 0.775390625, 0.82568359375, 0.123046875, -1.5556640625, 0.89892578125, -1.1982421875, 1.16796875, 1.1708984375, 0.73681640625, -0.6513671875, -0.042999267578125, 0.919921875, -0.264404296875, -0.69873046875, 0.315185546875, -0.435791015625, -0.93408203125, -0.25537109375, 0.0902099609375, 0.955078125, 0.09063720703125, -1.28515625, 1.4560546875, 1.216796875, -1.177734375, 0.3662109375, -0.7119140625, -0.437744140625, 2.388671875, 0.15673828125, -0.2291259765625, -0.1722412109375, 0.319580078125, 1.029296875, 1.3974609375, -1.30859375, 1.4345703125, 0.337646484375, 0.24365234375, 0.274169921875, -0.308837890625, -1.072265625, -0.366455078125, -0.27392578125, 0.6962890625, 1.443359375, -0.2607421875, -0.98681640625, 0.82666015625, -0.34814453125, 0.235595703125, 0.84619140625, -0.84130859375, 0.334228515625, 0.446044921875, -0.32470703125, -1.0322265625, 0.80078125, -0.0880126953125, 1.857421875, -1.1865234375, 0.27880859375, 0.75439453125, 0.1458740234375, 0.80908203125, -0.58984375, -0.54736328125, 2.328125, -1.2353515625, -1.31640625, 0.45556640625, -1.4443359375, 1.1552734375, 1.3310546875, -0.62548828125, 0.263916015625, 1.0869140625, -0.5107421875, 0.91357421875, -0.8564453125, -0.67333984375, 1.587890625, 0.57080078125, -0.8564453125, 0.1131591796875, 1.052734375, 0.1490478515625, -1.3291015625, -0.86328125, -2.833984375, -0.537109375, 0.75048828125, -0.2001953125, 1.166015625, 0.0679931640625, -0.55615234375, -0.75439453125, 0.408935546875, -0.034942626953125, -0.052032470703125, -1.109375, -1.2392578125, -0.0921630859375, -1.4619140625, 0.420654296875, -1.0263671875, -0.37841796875, -0.7177734375, 1.796875, -1.646484375, 0.55810546875, -0.8330078125, 0.52294921875, 0.494384765625, 0.090576171875, -1.0205078125, -0.215576171875, 0.47509765625, -0.366455078125, -0.642578125, -0.420166015625, -1.7197265625, -0.1458740234375, -0.383544921875, 0.035369873046875, -0.77099609375, 0.1925048828125, -0.49755859375, -1.33984375, -0.430419921875, 0.433837890625, -1.0087890625, -2.025390625, -0.837890625, 0.06719970703125, -2.2734375, 0.69287109375, 2.02734375, -1.13671875, -0.2012939453125, 0.9990234375, 1.9072265625, -0.7412109375, -0.1678466796875, -1.51953125, -0.78955078125, 0.611328125, 0.47900390625, -0.135009765625, 0.70703125, -0.56689453125, 1.884765625, -1.5478515625, 0.908203125, -0.87744140625, 1.55078125, -0.86376953125, -0.51513671875, 1.04296875, 0.65869140625, 0.1644287109375, 0.44384765625, 1.2333984375, 2.810546875, 1.1669921875, 2.259765625, 0.218505859375, -0.363037109375, 0.28564453125, -1.044921875, -0.53076171875, 0.1484375, 0.59619140625, 0.0026874542236328125, -0.701171875, -1.5400390625, -0.005786895751953125, 1.25, 1.4296875, -0.05682373046875, -1.349609375, -1.173828125, 1.3203125, -0.86474609375, -0.509765625, 0.734375, -0.76513671875, -0.666015625, 1.48828125, -0.0313720703125, 0.51953125, 0.64697265625, -2.318359375, -0.1177978515625, 0.75048828125, 0.2364501953125, -0.58740234375, -0.8232421875, 1.056640625, 0.379638671875, 2.19921875, -1.970703125, -1.0419921875, -0.28515625, 1.3203125, -0.6083984375, -0.014892578125, -0.3427734375, -0.264404296875, -1.2763671875, 0.1766357421875, 1.2197265625, -1.3701171875, -0.19189453125, -0.39453125, 1.3515625, -1.0498046875, -0.495361328125, -0.5419921875, 2.294921875, -0.78564453125, 0.27685546875, 1.0947265625, 0.609375, -0.63037109375, 1.326171875, 0.5966796875, 1.59765625, -0.8779296875, -0.8623046875, 0.2900390625, -0.56591796875, 1.4599609375, -2.140625, -0.0911865234375, -0.476318359375, 3.111328125, 0.32177734375, 1.6142578125, 0.1649169921875, -1.830078125, -1.0673828125, 2.1015625, -1.4140625, 1.7587890625, 1.171875, -0.368408203125, -2.087890625, 0.398681640625, 0.60302734375, 0.55322265625, -1.353515625, -1.310546875, -0.84423828125, 0.456298828125, -0.90185546875, 0.005992889404296875, -0.65966796875, -0.485595703125, -0.666015625, -0.032196044921875, 0.9951171875, 1.6298828125, -0.0926513671875, 0.472412109375, 1.0859375, 0.94482421875, -1.15625, -0.343017578125, 0.68310546875, -0.322509765625, -0.59814453125, -0.06658935546875, 0.059356689453125, 0.278076171875, 0.236572265625, -0.8466796875, 0.5244140625, 0.55078125, -0.51318359375, 1.3427734375, -0.56201171875, 0.51123046875, 0.740234375, 0.05670166015625, -0.137939453125, -0.3125, 2.203125, -3.236328125, -1.6767578125, -0.48095703125, 1.5595703125, 0.29248046875, -0.9755859375, 2.96484375, 0.2232666015625, -0.74267578125, 0.465576171875, -1.8291015625, -1.54296875, -0.247802734375, 0.0307464599609375, -0.55078125, 0.362060546875, 1.6396484375, 0.947265625, 0.626953125, -1.23828125, 0.56884765625, 0.2109375, 0.1270751953125, -0.7568359375, 1.822265625, 0.61181640625, 0.99658203125, 2.619140625, 0.89111328125, -1.59765625, 1.03515625, 0.50244140625, -0.513671875, -0.65869140625, -0.7802734375, -0.470458984375, 1.07421875, 0.412109375, 0.52978515625, -1.095703125, 0.354736328125, 0.01666259765625, -1.2177734375, 0.69384765625, -0.82080078125, -0.3994140625, 1.162109375, -0.1376953125, 1.689453125, 0.26708984375, -0.1739501953125, -1.1591796875, 1.0791015625, -0.2237548828125, 0.73828125, -1.4375, -0.442626953125, -0.53173828125, 0.16650390625, 1.4853515625, -1.326171875, 0.418701171875, 0.97900390625, 1.177734375, 0.392578125, 0.1907958984375, 0.6767578125, 0.07110595703125, -0.92333984375, 0.72119140625, -0.06768798828125, 0.445068359375, 1.7646484375, -0.87939453125, 0.12335205078125, 1.37890625, -2.150390625, -1.3505859375, -1.0859375, -0.049652099609375, -0.53125, 0.58154296875, 0.88134765625, -0.79296875, -1.09765625, -0.090576171875, 0.37353515625, 0.384033203125, -1.8076171875, 1.0146484375, 0.80859375, 1.03125, 1.931640625, 0.442138671875, 1.5869140625, 1.142578125, 3.458984375, 0.4228515625, 1.1630859375, 1.8544921875, 2.1796875, 0.43310546875, 0.341064453125, -0.50830078125, 0.65966796875, 0.12298583984375, -1.3544921875, -0.42578125, -1.7900390625, 0.02362060546875, -0.0122833251953125, 1.8779296875, -0.2998046875, -0.98583984375, -0.8603515625, -1.0537109375, 0.418212890625, -0.6240234375, 0.29833984375, -0.12890625, 1.029296875, 0.99072265625, -1.322265625, 0.521484375, 0.039398193359375, -0.81494140625, -1.6318359375, 1.5244140625, -0.11212158203125, 0.43603515625, -1.951171875, 0.283203125, -0.359375, 0.58740234375, 0.332763671875, -0.50830078125, 1.4599609375, -0.70361328125, -0.1363525390625, -1.3349609375, 0.65576171875, -0.10748291015625, 0.55859375, 0.63232421875, -0.01442718505859375, 0.347412109375, -0.385009765625, -0.388427734375, 0.2666015625, -0.281005859375, 0.27294921875, -0.9072265625, -1.1806640625, -0.2880859375, 2.908203125, -0.609375, -0.155029296875, 1.0166015625, 2.705078125, 0.9150390625, -0.0196380615234375, 0.323486328125, 1.3271484375, -1.212890625, -0.9375, 1.275390625, 0.6298828125, -0.51171875, 0.52880859375, -1.2216796875, -0.432373046875, -1.3671875, 1.48828125, -1.4453125, 1.396484375, 1.0048828125, -0.75146484375, -0.092529296875, -0.43017578125, 0.038360595703125, -0.129150390625, 0.31201171875, 2.259765625, -0.37939453125, -0.11761474609375, -0.356201171875, 1.5673828125, -0.98046875, -0.14111328125, -2.083984375, -0.51513671875, 1.4833984375, -1.455078125, 0.12744140625, 1.3623046875, -1.0859375, 1.125, 0.368896484375, -0.60302734375, 1.9130859375, 0.81298828125, 0.36572265625, 1.4169921875, 0.75439453125, 0.92431640625, 0.8984375, -0.184326171875, 0.372802734375, -0.475341796875, 1.162109375, -1.5888671875, 0.430908203125, 0.2015380859375, 1.291015625, 1.9482421875, -0.6123046875, 0.17041015625, 0.137451171875, -3.48828125, 1.6279296875, 0.90673828125, 1.6708984375, 0.5419921875, 0.5009765625, 1.216796875, 0.91943359375, -1.2197265625, 1.658203125, -1.6640625, 0.69384765625, 3.087890625, 1.3212890625, 0.240234375, -0.08441162109375, -0.4228515625, 0.783203125, -0.55322265625, 0.314697265625, 1.6923828125, -0.2298583984375, 0.26025390625, -0.481689453125, 0.0806884765625, 0.99658203125, -0.46484375, 2.0546875, 0.57861328125, -0.63037109375, 1.2314453125, -1.0068359375, 0.6943359375, 1.3984375, -2.244140625, 0.11614990234375, -1.3837890625, 0.13427734375, 0.9619140625, 1.77734375, -1.1923828125, 0.85693359375, -1.841796875, 0.111083984375, -0.7744140625, -1.4287109375, -0.09625244140625, 2.822265625, 1.0068359375, 0.1796875, -1.412109375, 1.400390625, 1.1064453125, 0.80322265625, 0.1871337890625, 0.28076171875, 1.0654296875, 0.337890625, -0.398193359375, 1.0087890625, 2.109375, 0.41064453125, 1.779296875, 0.287109375, -0.71826171875, 0.32861328125, 1.05859375, 0.90771484375, -1.4638671875, -0.57373046875, -0.42333984375, -1.0703125, -0.9404296875, 1.12109375, -0.08294677734375, -1.224609375, 0.427734375, -2.1640625, -1.046875, -1.45703125, 0.61181640625, 0.46484375, 1.078125, -1.0859375, -1.029296875, 0.259521484375, -0.73193359375, 0.76513671875, 0.603515625, -0.66845703125, -0.5361328125, -1.806640625, 1.5419921875, 0.82470703125, 3.302734375, 1.5126953125, 0.88037109375, -1.35546875, 1.2548828125, 0.230712890625, -1.6142578125, 0.2066650390625, -0.83251953125, -0.5361328125, -1.0888671875, 0.291015625, 2.392578125, -0.79638671875, -0.51953125, -0.11370849609375, 0.11279296875, -1.251953125, -0.6787109375, -0.74853515625, 0.10272216796875, -0.09124755859375, -0.703125, -0.61865234375, 0.248779296875, 1.1640625, 1.0537109375, 1.271484375, -0.401611328125, -0.2020263671875, -0.587890625, 1.2373046875, -0.7275390625, 1.814453125, -0.7392578125, 0.1075439453125, -0.50244140625, 0.021575927734375, 0.53759765625, -1.32421875, 0.283447265625, 0.61328125, -1.5615234375, 0.163818359375, -0.69482421875, 1.2236328125, -0.71337890625, 0.07855224609375, 0.904296875, 0.406005859375, -1.71875, -1.5087890625, 0.06634521484375, 1.408203125, -0.93359375, 1.3251953125, -0.08709716796875, 0.0723876953125, -0.423583984375, -0.61376953125, 0.828125, 1.6533203125, -0.775390625, 0.29931640625, 2.2109375, 0.61083984375, 0.33837890625, 0.298828125, 1.158203125, -0.0165557861328125, 1.416015625, 0.33837890625, -0.1859130859375, -1.2587890625, 0.79052734375, -2.365234375, 0.9072265625, 0.80078125, -1.291015625, 1.1201171875, -1.0390625, -0.410888671875, -1.705078125, 1.2548828125, 0.4912109375, 0.252197265625, -2.22265625, -0.5341796875, -1.1943359375, -0.274169921875, 1.119140625, 1.2890625, -1.583984375, 1.275390625, -0.49609375, 1.90625, 0.13330078125, 0.548828125, -0.6328125, -0.088623046875, -0.264404296875, -1.1005859375, 0.55126953125, -0.25048828125, -0.33935546875, -0.327392578125, -0.81689453125, 0.74658203125, -0.95361328125, 1.5078125, 0.8603515625, 0.07269287109375, -1.541015625, 0.1038818359375, 0.75244140625, 2.0, -2.27734375, 0.27978515625, 0.2254638671875, 1.0234375, 1.4287109375, 1.92578125, -1.1240234375, -1.1572265625, 0.8291015625, 0.1932373046875, 0.41943359375, -0.41455078125, -0.386962890625, 0.25537109375, 2.611328125, -2.04296875, 0.61181640625, -1.21875, -0.33349609375, 1.626953125, -0.81640625, -1.7138671875, 0.62890625, 1.796875, 2.92578125, 0.0234527587890625, -1.279296875, -0.56005859375, 0.08203125, 2.123046875, 0.370849609375, 1.068359375, 0.4775390625, -1.27734375, 0.9794921875, -1.9697265625, -0.245849609375, 0.65478515625, 0.125732421875, 1.1953125, 0.459228515625, 0.023468017578125, 2.673828125, -1.361328125, 1.4697265625, 0.046600341796875, -0.1151123046875, -0.0899658203125, -0.7861328125, -1.767578125, -2.359375, -0.26513671875, 1.552734375, -0.39306640625, -1.1435546875, 0.62109375, -0.2335205078125, -0.9443359375, 0.6689453125, -2.015625, -0.86376953125, -0.97900390625, 0.13330078125, 0.489013671875, -0.8662109375, 1.0107421875, 0.14501953125, -1.6298828125, 0.2744140625, 0.59521484375, 0.53515625, 0.8974609375, 1.2958984375, -1.509765625, 0.77978515625, 1.0634765625, 1.162109375, 0.64794921875, -0.49267578125, -0.6220703125, -0.7275390625, 0.25830078125, 0.1156005859375, -1.6708984375, 2.357421875, -0.447998046875, 0.409423828125, -0.00705718994140625, -1.3935546875, 1.5927734375, 0.41015625, -2.4375, 0.391845703125, -0.07318115234375, 0.163330078125, 0.62744140625, 1.83203125, 2.02734375, 0.0029468536376953125, -0.0276336669921875, 0.315673828125, 1.724609375, 0.108154296875, -2.3046875, 2.087890625, -1.1328125, -0.8056640625, 0.6513671875, -1.65234375, 0.58203125, -0.32861328125, -0.1778564453125, 1.3759765625, 0.97265625, 1.3603515625, -0.76220703125, -0.60400390625, -0.62841796875, -1.56640625, -2.1640625, 1.384765625, -0.457275390625, -0.405517578125, 0.708984375, -0.418212890625, -0.6318359375, -0.1871337890625, 0.80419921875, -1.5302734375, -1.1005859375, 1.0751953125, -0.202392578125, 1.1103515625, 0.384521484375, 0.98779296875, -0.7578125, -1.541015625, 0.96533203125, -0.0933837890625, 1.8046875, 0.82666015625, 1.408203125, -0.037689208984375, -1.6962890625, -0.09088134765625, 0.15185546875, -0.46142578125, 0.1170654296875, -1.1318359375, 1.97265625, 0.042144775390625, 0.3271484375, -1.6796875, 0.94287109375, 0.12164306640625, 1.158203125, 1.9892578125, -0.069580078125, -0.370361328125, -1.328125, 1.4609375, -0.358642578125, 1.4052734375, 1.4873046875, 0.12030029296875, -0.8798828125, 1.859375, -0.763671875, -1.427734375, -0.70849609375, 0.88330078125, -0.091552734375, -0.599609375, 0.430908203125, 0.759765625, 1.267578125, 2.802734375, -1.2509765625, -0.378173828125, -0.5615234375, -1.1796875, -0.09356689453125, 1.03515625, -2.18359375, -0.1790771484375, 1.349609375, 0.493408203125, 0.68896484375, 0.697265625, -0.411376953125, 0.0693359375, 0.08251953125, -1.44921875, -0.94970703125, 0.07403564453125, 1.4970703125, 1.1865234375, -0.7138671875, -0.494873046875, -1.306640625, 0.9111328125, -1.263671875, -0.22412109375, 0.32275390625, -1.0625, 0.27197265625, 0.98974609375, -0.85595703125, 1.4404296875, 0.10797119140625, 0.31396484375, -0.50537109375, 0.80712890625, -0.14501953125, -0.1097412109375, 0.053741455078125, 0.61474609375, 0.6708984375, -1.09765625, -0.15625, 0.278564453125, 0.55419921875, -1.7685546875, 0.61083984375, -0.138427734375, -0.2568359375, -1.904296875, 0.51611328125, -0.46728515625, 1.1396484375, -0.0121612548828125, 0.77294921875, -0.1876220703125, -0.9326171875, 0.95556640625, -2.677734375, 1.1337890625, -0.81787109375, 0.6103515625, -0.07958984375, -0.260986328125, 0.56982421875, 0.09149169921875, 1.5029296875, 1.822265625, -1.2275390625, 0.69970703125, -0.72119140625, -0.52685546875, 0.7734375, 0.609375, 0.83154296875, 0.1861572265625, 0.417236328125, -0.3935546875, -1.1474609375, -0.195556640625, 0.019622802734375, 0.9296875, -0.60205078125, 0.83984375, -1.798828125, 1.025390625, -0.53173828125, -0.845703125, -1.220703125, -0.3671875, 0.95263671875, -0.951171875, -0.55419921875, -0.10003662109375, 1.322265625, 1.333984375, -1.9638671875, 0.1573486328125, 1.5224609375, 1.1982421875, -0.9169921875, 0.413330078125, -0.31640625, 0.494140625, 0.2138671875, 1.2412109375, 0.400634765625, -0.11029052734375, -1.451171875, 0.2861328125, -0.89794921875, -0.40478515625, -0.380126953125, 0.4169921875, 2.365234375, 0.458984375, 1.306640625, 0.94482421875, 0.26806640625, 0.712890625, -1.845703125, 1.0751953125, -0.299072265625, -0.465087890625, -0.150390625, -0.95751953125, 0.7138671875, 0.1494140625, -0.83984375, 0.332763671875, -0.921875, -0.08843994140625, -0.386474609375, -0.24609375, -0.74462890625, 0.8681640625, 0.194580078125, 0.2374267578125, -0.1622314453125, -0.072509765625, 2.1171875, -0.662109375, 0.67041015625, -1.111328125, 2.1796875, -0.7138671875, 1.0927734375, -0.2288818359375, 0.446533203125, 1.0595703125, -0.339111328125, -0.39599609375, 0.326416015625, -0.54248046875, 0.05255126953125, 0.285888671875, 0.94189453125, 0.3388671875, 0.32421875, 0.293212890625, -0.759765625, 0.36962890625, 0.296630859375, 0.46875, 0.6044921875, -0.82666015625, -0.537109375, 0.57861328125, 0.806640625, 0.178466796875, 0.291259765625, 0.289306640625, 0.15673828125, -0.986328125, 0.6787109375, -0.2342529296875, -1.017578125, 0.1387939453125, -0.8720703125, 0.90185546875, 0.384521484375, 0.7412109375, -1.4248046875, 1.33984375, -0.465576171875, -0.8427734375, 0.161865234375, -0.81396484375, -0.15869140625, -1.7314453125, -0.927734375, 1.193359375, 0.6806640625, 1.419921875, -0.28759765625, -0.33935546875, -1.62109375, 0.18212890625, 1.2197265625, -0.6787109375, 0.304931640625, -1.0517578125, -0.3369140625, -1.0126953125, -0.016998291015625, 0.1697998046875, -0.7333984375, 1.1259765625, -0.73193359375, -0.6025390625, -0.309814453125, -0.0906982421875, -0.331787109375, -0.2080078125, -1.15234375, 0.1490478515625, -0.55908203125, -0.92626953125, -0.97265625, 1.2333984375, -0.1910400390625, 0.046356201171875, 1.2783203125, 1.3046875, -0.76806640625, 0.02001953125, -0.49267578125, -0.708984375, -0.2000732421875, 1.4248046875, -1.15625, -0.48828125, 0.359130859375, 0.62890625, -1.0380859375, 0.60400390625, -0.09149169921875, -0.73779296875, 0.1331787109375, 0.451171875, -0.86572265625, 1.787109375, 0.68408203125, 0.30859375, 0.140625, -0.578125, -0.6162109375, -0.20458984375, -0.350341796875, -1.404296875, 1.490234375, -0.9150390625, 0.30712890625, 0.3720703125, -1.2294921875, 0.140625, 0.53955078125, -0.74658203125, -0.36083984375, -2.646484375, 0.1839599609375, 0.11456298828125, -1.17578125, 0.7119140625, -1.2890625, 0.59130859375, -0.875, -0.04095458984375, 1.736328125, -0.71044921875, 0.134033203125, -0.350341796875, 0.962890625, -0.09283447265625, 0.15478515625, -2.0390625, -0.043487548828125, 0.67236328125, -0.486328125, 0.8994140625, 1.71875, -0.0899658203125, -0.77490234375, -1.8896484375, 0.91357421875, 0.06597900390625, -0.2607421875, 1.53515625, 0.023468017578125, -0.2216796875, 1.0654296875, 1.23828125, -0.82373046875, -0.66357421875, -1.169921875, 0.51123046875, -2.576171875, -1.615234375, 0.412841796875, -0.342529296875, 0.421875, -0.73583984375, -0.9375, -1.333984375, -1.2587890625, -0.93603515625, -0.1368408203125, -1.185546875, -1.4658203125, -0.9375, 1.01171875, -0.03997802734375, -0.1932373046875, 0.97705078125, 0.07025146484375, -0.1895751953125, 1.0712890625, 0.98974609375, 1.4921875, -0.068603515625, -0.052764892578125, -2.41015625, -0.7626953125, -1.021484375, 0.34912109375, -0.10980224609375, -0.62939453125, -0.054046630859375, 0.70458984375, -0.546875, 0.07672119140625, 0.412353515625, 0.87109375, -0.57080078125, -1.2734375, -0.64697265625, 0.76708984375, 0.47900390625, -0.4423828125, -0.8212890625, -1.5673828125, -1.048828125, -0.79052734375, 1.8408203125, -2.330078125, -0.30859375, 0.271728515625, -0.11822509765625, 1.3359375, 0.1717529296875, -0.73974609375, 0.342529296875, 0.61328125, 0.88818359375, -2.177734375, -0.5126953125, -1.4619140625, -1.400390625, 0.78076171875, -0.86181640625, -0.419189453125, 2.791015625, -2.16796875, 0.455810546875, -0.50439453125, 1.193359375, 1.0849609375, 1.3466796875, -2.583984375, -0.033905029296875, 0.7421875, 0.382568359375, 1.6884765625, 0.84375, 0.391357421875, 0.74951171875, 1.1416015625, 1.361328125, -0.955078125, -0.55322265625, 0.349853515625, -0.96337890625, 0.345458984375, 1.150390625, 0.81005859375, 0.62255859375, -0.6962890625, 0.0947265625, 0.052215576171875, -1.1953125, -0.362548828125, 0.293701171875, 0.7978515625, -0.0205841064453125, 2.02734375, 1.068359375, 2.74609375, -1.23046875, 0.0350341796875, -1.107421875, -0.6630859375, -0.317138671875, -1.298828125, 1.43359375, 0.24267578125, -1.25390625, 0.313720703125, 0.505859375, 0.08612060546875, 1.599609375, -0.3974609375, -0.3212890625, 0.071044921875, -2.205078125, 0.345703125, -1.1982421875, 0.5703125, -0.892578125, 0.125732421875, -0.265625, -0.01343536376953125, -1.0107421875, 2.333984375, 0.65625, 0.6552734375, -0.20263671875, -0.28662109375, 0.9072265625, 0.355712890625, 0.7685546875, -0.57958984375, -0.52587890625, -0.294189453125, 1.34375, -0.25927734375, -0.7431640625, 0.1376953125, -0.1136474609375, 0.199951171875, -0.22314453125, 1.06640625, -0.55615234375, 0.409912109375, 0.580078125, 0.92041015625, -0.11114501953125, 0.654296875, -1.5283203125, -0.222412109375, -0.408935546875, 0.986328125, -1.3779296875, 0.87353515625, 0.10650634765625, -0.67626953125, -0.16455078125, -1.0966796875, -0.7158203125, 0.7392578125, -1.8974609375, -0.6650390625, -1.509765625, -1.2333984375, 0.576171875, 0.986328125, -1.177734375, 0.160888671875, 0.0205841064453125, -0.2330322265625, -0.55908203125, -0.1572265625, -0.477783203125, -1.130859375, 0.383544921875, 0.5166015625, -1.7783203125, -0.53515625, -0.3798828125, 2.212890625, 1.84375, -0.39453125, -0.20068359375, -0.0247344970703125, -0.33154296875, -1.4697265625, 0.288330078125, -0.662109375, -0.8427734375, -0.0811767578125, 2.4765625, 2.23828125, -1.1064453125, 1.490234375, -0.307861328125, -0.73583984375, 0.6455078125, -0.73193359375, -2.775390625, -0.56103515625, 2.171875, 0.50439453125, 1.5048828125, -2.017578125, 0.318359375, -0.98974609375, -0.74951171875, 1.0205078125, -0.361328125, -3.27734375, -1.4404296875, -0.5693359375, -0.5634765625, 2.142578125, -1.630859375, 1.8828125, -0.9423828125, 1.2666015625, -0.448486328125, -0.4384765625, -1.4970703125, -1.1962890625, -0.82666015625, 1.59375, -0.206787109375, -0.4853515625, -0.04656982421875, 0.3115234375, 1.609375, 1.2421875, -0.358642578125, -0.0195465087890625, -0.08782958984375, -0.8251953125, 0.484619140625, -0.370361328125, -1.3525390625, -0.22802734375, 0.1522216796875, 0.2491455078125, -0.449462890625, -0.415771484375, 0.115966796875, -1.0185546875, 0.38037109375, 0.74658203125, -0.7060546875, 0.481689453125, 2.232421875, 0.41015625, -0.88330078125, 1.0791015625, 1.103515625, 0.455322265625, 0.8642578125, -1.5146484375, 2.158203125, -0.414794921875, -0.0997314453125, 1.83203125, -1.00390625, -0.61181640625, 1.076171875, 0.5888671875, 0.1353759765625, -0.66552734375, 0.0723876953125, 0.337158203125, 0.70751953125, 0.343017578125, 0.55224609375, -0.484619140625, 0.08013916015625, -1.404296875, 0.374755859375, -0.41455078125, -0.58251953125, 0.80322265625, -0.1748046875, 0.48583984375, -1.0849609375, 0.343017578125, -0.32568359375, -1.4892578125, -0.392822265625, -0.062408447265625, 0.98388671875, -0.48974609375, -0.81494140625, 0.64208984375, 0.69189453125, 0.17626953125, -0.5927734375, 1.29296875, -0.1748046875, -2.095703125, -0.6416015625, 0.136474609375, 0.1656494140625, 0.8623046875, 1.3876953125, 1.2705078125, -1.224609375, -2.0546875, 0.97900390625, -0.80078125, -0.2410888671875, -0.323974609375, -0.67626953125, -0.28662109375, 1.6064453125, 0.53125, 1.1787109375, -0.385498046875, 0.58984375, -0.70556640625, -0.45068359375, -1.7841796875, -0.01922607421875, 0.88525390625, -1.5400390625, 1.3955078125, 0.427734375, -0.1802978515625, -1.482421875, 2.08203125, -1.09765625, 1.4130859375, -1.1708984375, 0.2327880859375, 1.45703125, -0.42578125, -1.6767578125, 0.90087890625, -0.2052001953125, -0.59765625, -0.4609375, -0.80322265625, 0.11505126953125, -0.73876953125, -1.1328125, -1.0126953125, -1.146484375, -0.2197265625, -0.57177734375, -0.82666015625, -0.27099609375, 1.658203125, 0.11932373046875, -0.82763671875, 0.49365234375, 2.185546875, 0.6796875, 1.2470703125, 0.6572265625, 0.34326171875, -0.908203125, -0.27978515625, -1.1103515625, 0.250732421875, 0.6845703125, 0.56494140625, 0.37255859375, 0.880859375, 0.787109375, 0.50390625, 0.50341796875, -0.290283203125, 2.4375, -0.9638671875, 0.5537109375, 0.2210693359375, 0.51513671875, 0.322509765625, 0.9267578125, 1.1123046875, -0.369140625, -0.385009765625, -0.61767578125, 1.912109375, 0.05230712890625, -0.5234375, 0.0216827392578125, 0.49072265625, 0.450439453125, 0.197998046875, 1.3828125, 0.8759765625, 0.1370849609375, 0.25537109375, 1.67578125, -0.806640625, -1.3232421875, -0.74072265625, 1.10546875, 0.480712890625, -1.220703125, 0.2666015625, 0.6162109375, 0.0240478515625, -1.7626953125, -0.1158447265625, -0.9423828125, -0.9345703125, 0.5703125, -0.705078125, 1.8173828125, -1.2373046875, 0.333984375, 0.8447265625, -2.064453125, 1.298828125, -1.369140625, -0.99365234375, 0.01032257080078125, 0.379638671875, -0.32177734375, -0.1663818359375, -0.3984375, 0.388427734375, -0.12091064453125, -0.029022216796875, -0.2088623046875, 0.51123046875, 0.0267486572265625, -0.268310546875, -1.390625, 0.8203125, -1.0029296875, -1.806640625, -0.62744140625, 1.0302734375, 1.16015625, -0.01337432861328125, 0.6708984375, 0.650390625, -0.216552734375, -0.01123809814453125, 1.259765625, 0.38525390625, -1.1123046875, -1.21484375, -0.5263671875, 0.34326171875, 1.6865234375, -0.265625, 1.2041015625, 0.13916015625, 0.46142578125, 0.490234375, -0.1351318359375, -0.41650390625, -0.79931640625, -0.373291015625, 1.630859375, -0.218017578125, -0.29296875, -1.0859375, -0.284912109375, 1.2265625, -1.455078125, -1.650390625, -0.892578125, 0.7021484375, 0.479248046875, 2.39453125, 1.017578125, 0.82421875, 0.09857177734375, 0.343505859375, 0.77197265625, -0.2379150390625, -0.432861328125, -0.53662109375, -0.318115234375, 0.038665771484375, 0.51611328125, -2.130859375, 1.4013671875, -0.18115234375, -0.43701171875, -2.33984375, 0.073486328125, -2.244140625, 0.8818359375, 0.494873046875, -0.98876953125, 0.40478515625, -0.67578125, -0.221435546875, -0.8369140625, 0.267578125, -0.912109375, 1.4150390625, 0.047760009765625, 0.450927734375, -0.2734375, -0.91015625, 0.466552734375, -0.488037109375, -2.126953125, -0.861328125, 0.8076171875, -0.145751953125, 0.307373046875, 1.1201171875, 0.751953125, 0.38037109375, -1.0947265625, 0.131103515625, -0.93896484375, -0.32373046875, 0.794921875, -0.26025390625, 1.740234375, 0.357666015625, -2.048828125, 1.48046875, -0.572265625, 0.481689453125, 0.515625, 0.1966552734375, -1.0146484375, 0.4873046875, -1.4833984375, -0.6142578125, -0.70068359375, -0.1541748046875, 1.9189453125, -2.509765625, 0.279052734375, 1.2314453125, -1.2607421875, -0.0221405029296875, -0.1806640625, -0.078125, -0.41259765625, 0.029998779296875, 0.556640625, -1.12109375, 1.02734375, 0.58984375, -0.37890625, 1.416015625, 0.18115234375, -1.927734375, -0.12481689453125, 0.2138671875, 0.63525390625, 0.55908203125, -0.81103515625, -0.58740234375, -0.68896484375, -0.92724609375, 0.4951171875, 1.46484375, 0.08270263671875, -0.958984375, 1.982421875, 0.09356689453125, 1.3740234375, -0.82177734375, 0.136962890625, -0.1343994140625, -0.31005859375, 1.0849609375, -0.357666015625, 1.13671875, -0.96923828125, 0.5078125, 2.4609375, -1.5400390625, -0.10455322265625, 0.337158203125, -0.0341796875, -0.5986328125, -1.6708984375, -0.16357421875, 0.1932373046875, -0.043426513671875, 1.5810546875, 1.8310546875, 1.8203125, 0.343994140625, -1.71484375, -1.7138671875, -1.1962890625, -0.55908203125, -1.1953125, 1.6943359375, 1.1142578125, 0.71533203125, -0.8193359375, -1.123046875, 1.25390625, 1.3935546875, 0.0675048828125, -0.006969451904296875, 0.13037109375, 1.544921875, -0.7158203125, 0.9833984375, 0.1585693359375, 0.736328125, 0.442138671875, -1.029296875, 0.478515625, -1.2060546875, 2.72265625, 0.57177734375, -1.453125, -0.0241851806640625, -1.8369140625, 1.2119140625, -2.224609375, 1.0498046875, 1.8896484375, 0.339599609375, -0.256103515625, 2.04296875, -0.615234375, -1.375, -0.354736328125, 0.56591796875, -0.398193359375, 0.328369140625, 1.357421875, -0.72607421875, -1.0185546875, -0.049774169921875, 0.1898193359375, 1.2529296875, -1.4267578125, 1.3408203125, 1.400390625, 0.40478515625, -1.53125, 0.1634521484375, 1.25, 0.58837890625, 1.2109375, -1.1591796875, -1.5546875, 0.28466796875, -0.52099609375, 0.1318359375, -1.0810546875, 0.12548828125, 0.59716796875, 1.3017578125, 1.32421875, -0.276611328125, -0.78173828125, -1.0947265625, 0.57861328125, -1.2978515625, -0.1446533203125, -0.552734375, -0.99267578125, 0.8779296875, 1.1513671875, 0.970703125, 1.921875, 0.436279296875, -1.2939453125, 1.37109375, 0.05792236328125, -0.1964111328125, -0.8984375, -0.9736328125, -0.49267578125, -0.54345703125, 0.2147216796875, -0.08221435546875, -1.9951171875, -0.66845703125, 1.134765625, 0.1007080078125, 1.521484375, 0.289794921875, -1.140625, -0.2154541015625, 2.25, -0.384033203125, 0.74609375, 0.385498046875, 3.044921875, 0.318603515625, 0.55810546875, -1.08203125, 0.283203125, 1.671875, -0.37353515625, -0.67138671875, -0.447265625, -2.9296875, 1.5390625, -0.251708984375, 1.5048828125, -0.634765625, 0.38330078125, -0.32080078125, -0.63232421875, -0.10589599609375, 0.12445068359375, -2.123046875, -1.2685546875, 0.48583984375, -2.111328125, -0.45361328125, 1.6328125, 0.53515625, 0.74853515625, 0.94677734375, -1.5302734375, 0.286376953125, 1.572265625, -0.075927734375, 1.125, 1.4306640625, 0.5810546875, -1.216796875, -1.328125, 0.80615234375, 1.216796875, -0.36376953125, 2.1171875, -1.6259765625, -0.5966796875, 0.226806640625, 0.263916015625, 0.22509765625, -1.064453125, -0.71484375, 0.00738525390625, -2.216796875, 1.755859375, -1.34375, 0.07366943359375, 0.11279296875, 0.051361083984375, -0.55908203125, 0.3125, 0.814453125, -1.2294921875, 0.48388671875, 0.57666015625, 1.1484375, 1.81640625, 0.81103515625, 1.234375, 0.56396484375, 1.46875, 0.20947265625, 1.568359375, -2.2890625, 0.10113525390625, -0.0278472900390625, -0.490234375, -2.4921875, -0.45849609375, 0.646484375, -0.114013671875, 0.826171875, 0.059906005859375, -0.328857421875, 0.10321044921875, -0.50244140625, 0.44775390625, -0.371826171875, -0.2666015625, -0.10986328125, -1.4853515625, 0.08697509765625, -1.4755859375, 1.2744140625, -0.323974609375, -1.990234375, 0.87060546875, 0.669921875, 0.44921875, 0.434814453125, -0.318603515625, -0.124755859375, -0.63427734375, -1.6015625, -1.09765625, -0.3740234375, -0.78173828125, -0.10186767578125, 0.72314453125, -1.41015625, -0.607421875, -0.06768798828125, 0.0047760009765625, -1.65625, 1.0517578125, 0.138916015625, 1.177734375, -0.6279296875, -1.408203125, -0.88720703125, -0.8388671875, -0.5751953125, 1.3916015625, -0.9267578125, 0.67822265625, 0.47509765625, -0.546875, 0.0435791015625, 0.955078125, -3.11328125, -0.54296875, -2.88671875, -0.15673828125, -0.7021484375, 0.136962890625, 1.4384765625, 1.6953125, -0.474609375, -0.2034912109375, -0.99853515625, 1.5576171875, -0.07550048828125, -0.472412109375, 2.345703125, 0.270263671875, -0.2408447265625, -0.413330078125, 0.346435546875, 0.302978515625, -1.5283203125, -0.47900390625, -0.5302734375, 0.47021484375, 0.44189453125, -1.375, 1.1376953125, -0.402587890625, -0.441162109375, -2.51953125, -1.42578125, 1.0869140625, -0.2060546875, 0.7392578125, -0.30322265625, 0.6416015625, 0.181396484375, 0.308349609375, 0.5849609375, 0.5419921875, 1.869140625, 0.8759765625, -1.576171875, 1.1201171875, -1.7021484375, 2.0078125, -0.5439453125, 0.71142578125, -0.14306640625, 1.7744140625, 1.1650390625, -0.290771484375, -0.226318359375, -1.271484375, 0.62646484375, 0.1988525390625, -0.6875, -0.334716796875, 0.08099365234375, 1.3525390625, -1.119140625, -0.8359375, -0.78662109375, -0.172607421875, 1.3115234375, -0.26708984375, -1.5380859375, -0.8369140625, 1.6865234375, 0.11590576171875, 0.85400390625, -1.501953125, 0.3076171875, 0.9951171875, -1.8330078125, 2.2421875, 0.6923828125, -0.9296875, 0.212158203125, 1.0908203125, 0.97021484375, 1.1005859375, -1.431640625, 0.5380859375, -0.2249755859375, -0.7099609375, -0.39501953125, -0.1219482421875, -0.14599609375, 1.197265625, -0.2095947265625, 0.806640625, 1.615234375, 1.9541015625, 0.256103515625, -0.341064453125, 0.5556640625, 1.2353515625, -3.072265625, -0.44921875, 0.74560546875, 0.238525390625, -0.1556396484375, -0.1949462890625, -0.74951171875, 0.11737060546875, 1.6337890625, -2.134765625, -0.9521484375, -0.440185546875, 0.47900390625, 0.337646484375, 0.013519287109375, -0.5029296875, -0.84423828125, -1.0693359375, -1.017578125, 0.78759765625, 1.4736328125, -0.337646484375, 0.62109375, -0.31298828125, 0.33349609375, 1.919921875, -0.372314453125, 1.1171875, -1.298828125, -2.10546875, 0.10589599609375, 0.1903076171875, 1.134765625, -1.0048828125, -1.8125, -0.97509765625, 0.83984375, 0.5908203125, -1.2744140625, 0.5888671875, -3.115234375, -0.94140625, 1.0537109375, -0.00424957275390625, 0.76220703125, 0.94921875, -2.01171875, 0.452880859375, -0.192138671875, 0.77880859375, -1.27734375, -1.2568359375, 0.56787109375, -0.73193359375, -1.2021484375, -0.418701171875, 0.056732177734375, -0.935546875, 0.716796875, -0.76123046875, -0.36865234375, -1.541015625, 2.517578125, -1.9072265625, 1.326171875, 0.7958984375, 0.69189453125, 0.67333984375, 0.57568359375, -0.2349853515625, 1.2412109375, -0.64697265625, -0.2088623046875, 1.0556640625, 2.125, 0.63623046875, -1.4990234375, 0.9677734375, -0.364013671875, -0.1273193359375, -0.09906005859375, 2.10546875, -1.26171875, 0.95068359375, -0.4443359375, -0.46728515625, -1.603515625, 0.62841796875, -0.28466796875, -0.089599609375, 0.410888671875, 1.005859375, -0.88916015625, -0.031524658203125, 2.765625, 0.8740234375, 0.529296875, 0.1087646484375, -0.91845703125, -0.53857421875, -0.444580078125, -1.15234375, 0.9384765625, 1.798828125, -1.2607421875, 1.041015625, 1.16015625, 0.2227783203125, -1.2421875, -1.142578125, -1.3564453125, 0.7841796875, 0.2362060546875, 0.30419921875, -1.44921875, -1.5048828125, 0.389404296875, -1.0869140625, -2.0703125, 1.0849609375, 1.1787109375, -0.41845703125, 0.86279296875, -0.609375, 0.0736083984375, -0.64404296875, -0.51708984375, 2.83203125, 0.048797607421875, 0.39599609375, 1.7978515625, -3.302734375, -0.9541015625, 0.53759765625, -0.073486328125, 0.4453125, 0.1292724609375, -1.2802734375, -0.24560546875, 0.26171875, -2.00390625, 0.78466796875, 0.09490966796875, 0.7255859375, -0.092041015625, -0.341552734375, -1.2080078125, 0.90869140625, -0.69189453125, 0.259521484375, 0.97705078125, 0.0843505859375, -1.125, -1.7919921875, -0.240966796875, -0.136962890625, -1.12890625, -0.287109375, 0.4267578125, -0.13623046875, 0.8212890625, -0.0279998779296875, -1.1875, -0.703125, -0.63671875, 0.4541015625, 1.3232421875, -1.58203125, -0.8916015625, 0.297607421875, -0.09356689453125, -0.0518798828125, 1.4404296875, -0.68505859375, -1.32421875, 0.29931640625, -2.181640625, -0.150634765625, -0.289306640625, -0.311279296875, 0.36962890625, -0.111328125, -1.5703125, 0.46484375, -0.414794921875, 1.1044921875, -1.3408203125, 0.54443359375, 0.1268310546875, 0.55224609375, -0.0012292861938476562, 0.134521484375, -1.3173828125, -1.7265625, -0.6328125, -0.419921875, -1.45703125, -0.7861328125, -0.5615234375, -0.92333984375, 1.3642578125, 0.2386474609375, -1.0771484375, -0.6923828125, 0.86865234375, -0.56103515625, -0.175048828125, -0.0061492919921875, -1.3427734375, 0.09075927734375, -0.626953125, -0.283935546875, -0.9580078125, 0.01447296142578125, -1.4345703125, 1.8623046875, -1.623046875, -0.1043701171875, -1.9296875, -0.7294921875, -1.6669921875, 1.7314453125, -0.1888427734375, -0.1346435546875, -0.21484375, -1.0986328125, 0.26953125, 1.517578125, -2.234375, 0.277587890625, -0.99755859375, 1.6025390625, 1.9970703125, 1.3134765625, -0.5712890625, 0.062225341796875, -1.015625, 1.1240234375, 0.128173828125, -0.521484375, 0.17626953125, 2.076171875, 1.3486328125, -0.0302734375, -1.8935546875, -0.30078125, -0.059722900390625, -1.2099609375, -0.7265625, 0.52880859375, 1.86328125, 1.5595703125, 0.67236328125, -1.5009765625, -0.359130859375, 1.341796875, 0.10198974609375, 1.0703125, 0.1365966796875, 0.9091796875, 0.2225341796875, 0.1953125, -0.404052734375, 0.1131591796875, -0.044952392578125, -0.90625, 0.23388671875, 0.72314453125, 0.449951171875, 0.2364501953125, -2.029296875, 0.615234375, -0.07818603515625, -0.8154296875, 0.701171875, -0.11376953125, 0.87060546875, -0.09075927734375, 0.01861572265625, 0.31103515625, 0.05859375, -0.4287109375, -0.833984375, -0.34423828125, 0.74853515625, -0.9599609375, 0.10931396484375, 0.1446533203125, -0.312255859375, -0.5810546875, 0.63916015625, -0.130859375, -0.8310546875, 0.3310546875, 0.94775390625, 0.2078857421875, -0.306884765625, -0.62060546875, 0.50048828125, 0.33056640625, -0.09649658203125, -0.392578125, -1.09765625, 0.88037109375, 0.64404296875, 0.269287109375, -0.6630859375, 0.07257080078125, 1.6357421875, -0.90478515625, 0.6767578125, -0.67041015625, -0.1888427734375, -0.5703125, -1.279296875, -0.517578125, -0.1441650390625, 1.0908203125, -0.1439208984375, -0.88671875, -3.41015625, -0.4482421875, -0.89501953125, -0.9091796875, 0.26513671875, -0.80126953125, -1.869140625, 0.77734375, -0.7705078125, -0.1435546875, -0.33251953125, 0.203125, -0.2159423828125, -0.469482421875, 0.471435546875, -1.0078125, -0.397705078125, 0.3662109375, -0.962890625, -0.0867919921875, -0.89453125, -0.7568359375, -0.67041015625, 0.11627197265625, 0.66845703125, 0.307861328125, 0.451904296875, -0.92529296875, 0.04541015625, -1.05078125, 1.9296875, 0.1689453125, 0.08038330078125, 1.1865234375, -0.316162109375, -0.548828125, -0.385498046875, -0.85107421875, -0.98876953125, 0.1138916015625, -1.5166015625, -0.84521484375, -0.0236663818359375, 0.23779296875, 0.57421875, 0.4501953125, -0.029693603515625, 0.71142578125, -0.673828125, 1.0849609375, -0.97216796875, -1.7197265625, -0.84521484375, 0.408447265625, -1.16796875, 1.6494140625, -0.7099609375, -0.4921875, 0.912109375, 0.90087890625, 0.544921875, 0.2178955078125, 2.3359375, -0.33447265625, -1.0478515625, 1.21484375, -1.615234375, -0.05670166015625, -0.378173828125, 0.2261962890625, 0.29931640625, -1.125, 0.0253143310546875, -1.4638671875, -0.149169921875, 0.178466796875, 0.5478515625, 0.728515625, -0.8779296875, -1.3671875, -0.55712890625, -1.7041015625, -0.368896484375, -0.5029296875, 1.091796875, -0.2230224609375, 0.4931640625, 0.9541015625, 0.2471923828125, -0.591796875, -0.33642578125, -0.493408203125, -0.8955078125, 0.307373046875, -0.79052734375, 0.51904296875, -1.341796875, -0.84423828125, -0.66357421875, 0.20947265625, -2.28515625, 0.3173828125, -1.587890625, 1.6171875, 0.99072265625, -0.54736328125, 0.241455078125, -1.0283203125, -0.52197265625, 0.53515625, 0.15966796875, 0.1683349609375, -1.6513671875, -0.1123046875, 0.89599609375, -0.372802734375, -0.387939453125, -0.3232421875, 0.052764892578125, 1.32421875, 1.072265625, -1.4287109375, -0.88671875, -0.7529296875, -0.12841796875, 1.4111328125, -1.3115234375, -0.7373046875, 0.004520416259765625, -1.501953125, 0.6474609375, -1.4609375, -0.49169921875, 0.64306640625, -0.2095947265625, -0.4169921875, 0.259765625, 1.146484375, 0.44140625, -0.1702880859375, -0.344970703125, 0.1927490234375, 1.0185546875, 0.416748046875, -0.203857421875, 1.9501953125, 0.369384765625, -0.1683349609375, 0.52685546875, 0.083740234375, -1.2451171875, -1.962890625, -0.85888671875, 1.0849609375, 0.43017578125, 1.5556640625, -0.65673828125, 0.39501953125, -0.7607421875, 2.361328125, 1.7734375, 0.91650390625, -0.9892578125, 0.51220703125, -1.4970703125, -1.9765625, -0.2257080078125, -0.54345703125, -0.15234375, 0.490234375, -1.6123046875, 1.2109375, -0.9423828125, -1.5546875, -0.1307373046875, -1.1455078125, 0.5888671875, -1.0537109375, -1.361328125, 0.1397705078125, -0.43310546875, -1.62109375, -0.7587890625, -0.019744873046875, -0.498779296875, -0.3818359375, -1.2783203125, 1.94140625, 3.21484375, -2.822265625, 0.0596923828125, 0.3193359375, -2.46484375, -0.06378173828125, 1.12890625, -1.2099609375, -1.1953125, -0.336669921875, 0.59912109375, -1.603515625, -0.583984375, -0.52490234375, 1.533203125, 0.430908203125, 0.54248046875, -0.51025390625, -1.0771484375, 1.3447265625, -0.3701171875, -0.411865234375, -0.72314453125, 0.1923828125, -0.58935546875, 0.1817626953125, 0.70654296875, 0.3642578125, -0.69677734375, -0.1025390625, 0.11138916015625, -1.4541015625, 0.060699462890625, -0.2354736328125, 1.1953125, -0.486328125, 0.0304412841796875, -0.278076171875, -1.1044921875, 0.75390625, -1.3232421875, 1.7255859375, -0.262451171875, 0.50830078125, -1.046875, -0.89697265625, -0.88037109375, 1.076171875, -1.076171875, 0.41748046875, -0.64111328125, 1.056640625, -0.54345703125, 0.78076171875, -1.3515625, 0.1358642578125, -0.256591796875, 0.485107421875, 0.4189453125, 0.5810546875, -1.2451171875, -1.0283203125, -0.64013671875, -1.2578125, -1.115234375, 1.228515625, 1.6162109375, 1.244140625, -2.5234375, -0.039398193359375, 1.458984375, 1.537109375, -0.471435546875, 1.564453125, 0.77392578125, -0.69287109375, -0.322265625, -1.1904296875, 0.491943359375, 2.38671875, -1.8876953125, -0.474853515625, -0.98681640625, 0.955078125, -0.1307373046875, -0.27392578125, -0.264892578125, 0.495361328125, 1.0908203125, 0.420654296875, -0.33984375, -0.71240234375, 2.98828125, 0.69287109375, -0.1947021484375, 1.615234375, 0.27978515625, 0.630859375, 0.1680908203125, 2.607421875, -0.252685546875, 0.78173828125, -1.4404296875, -0.37255859375, 1.1083984375, 2.267578125, 0.43603515625, -1.185546875, 1.2119140625, 1.9853515625, 1.921875, -0.052520751953125, 0.1854248046875, -1.53125, 0.88037109375, -1.1533203125, -0.86279296875, 0.2000732421875, 1.544921875, -1.296875, 1.19921875, -0.494140625, 0.83642578125, -1.5078125, 1.0029296875, 1.404296875, 1.9609375, -1.6875, 0.09173583984375, 0.0816650390625, 0.049774169921875, 1.1728515625, -1.4755859375, 1.8134765625, -0.2685546875, -1.4208984375, 0.0035724639892578125, 0.94091796875, 1.2744140625, -1.4755859375, -0.80712890625, -0.3916015625, 0.309814453125, 0.060791015625, -0.83544921875, 0.71826171875, 0.69580078125, -0.0106964111328125, -0.62158203125, 0.464599609375, 2.701171875, 0.55810546875, -0.9072265625, 1.849609375, 1.5849609375, -0.9150390625, -1.5771484375, 1.5234375, -1.791015625, -0.90869140625, 0.81640625, -1.6943359375, -0.31005859375, -0.89013671875, -0.15966796875, 0.08111572265625, 3.12890625, 0.2366943359375, -1.345703125, -1.1015625, 1.01953125, -1.1513671875, -1.248046875, 0.85693359375, 2.40625, 0.8681640625, -0.79541015625, -2.3828125, -0.396728515625, -1.5146484375, -0.630859375, 0.513671875, 0.4990234375, 0.57470703125, -0.796875, 1.3818359375, -0.2469482421875, 0.393798828125, -0.560546875, 0.4716796875, 0.6494140625, 0.3994140625, 0.88330078125, 0.146728515625, -1.2734375, 0.07220458984375, 0.5732421875, -0.379638671875, -0.09735107421875, -0.62646484375, 0.50390625, 0.272216796875, 0.60205078125, 0.351318359375, 0.6787109375, -0.145751953125, 2.017578125, -0.0443115234375, 1.2763671875, -0.5224609375, 2.947265625, -1.1748046875, 1.8173828125, 0.423828125, -1.4580078125, -1.97265625, -0.419921875, 0.5537109375, 0.4970703125, -0.0149383544921875, 0.81787109375, -0.4560546875, -1.251953125, 1.259765625, 0.7685546875, 0.8662109375, -0.92333984375, 0.7998046875, -1.5234375, 0.075439453125, 0.087890625, 0.2415771484375, 1.4267578125, 0.63232421875, 0.95751953125, -2.03125, -0.4560546875, 0.1573486328125, 1.408203125, 0.241455078125, -0.59033203125, 0.80322265625, -0.296630859375, -0.4716796875, -1.017578125, -0.243896484375, -0.91064453125, 1.373046875, -0.3271484375, 1.2939453125, 1.041015625, 2.251953125, -1.453125, 0.312744140625, -1.2578125, -0.78515625, -1.001953125, -1.8251953125, 0.90185546875, -0.317138671875, -1.7265625, 0.53369140625, 0.3603515625, 1.3486328125, -0.7568359375, -0.68310546875, 0.5859375, -0.982421875, 0.5888671875, -1.5830078125, 0.64892578125, -2.365234375, -0.95263671875, -0.61083984375, -0.06695556640625, -0.1610107421875, 0.087646484375, 0.366455078125, 0.7958984375, 0.38818359375, -0.5263671875, -0.0202178955078125, -1.1767578125, -1.576171875, -0.2305908203125, -0.8466796875, -1.0302734375, -0.56005859375, -1.2255859375, 1.240234375, 1.0009765625, 2.54296875, 0.82666015625, -1.8671875, 0.630859375, -0.91064453125, -0.260498046875, 0.223388671875, -0.462158203125, -0.11846923828125, 0.38232421875, 0.88525390625, 0.2161865234375, 1.1376953125, -0.74169921875, -0.2080078125, 0.357666015625, -0.3955078125, 0.06549072265625, 1.17578125, -0.6298828125, 1.9423828125, -0.232177734375, -0.70361328125, 0.052459716796875, -0.486328125, 0.465087890625, 1.7353515625, -0.043548583984375, 0.479248046875, -0.47802734375, -1.814453125, 1.626953125, 1.578125, -0.8369140625, 1.8251953125, -0.73583984375, -0.66064453125, 0.66259765625, -0.814453125, 1.1787109375, 0.9990234375, 1.666015625, 0.0357666015625, 2.166015625, -2.193359375, -0.32861328125, 0.71240234375, 0.07342529296875, -1.8056640625, -0.2222900390625, 0.10150146484375, -0.78271484375, -0.63037109375, 1.0205078125, 0.2919921875, -0.473876953125, -1.4619140625, -0.0655517578125, -1.44140625, 0.2117919921875, 0.01558685302734375, -0.044647216796875, -0.5546875, 0.298095703125, 0.404296875, -0.73583984375, 1.8173828125, -1.2685546875, 1.2275390625, 1.32421875, -0.0523681640625, 0.344482421875, 1.056640625, -1.0654296875, 0.68701171875, -0.48486328125, -0.458984375, -0.7880859375, 0.1895751953125, -0.2059326171875, -0.0299530029296875, -1.1748046875, -0.1185302734375, 1.103515625, -0.452392578125, 0.1815185546875, 0.9912109375, 0.485595703125, 0.56689453125, -0.434814453125, -0.28369140625, 1.0458984375, 1.1591796875, -0.6025390625, 1.984375, 0.9892578125, 1.1240234375, 0.90234375, -0.487548828125, -1.14453125, 2.556640625, -0.6796875, 0.1356201171875, -0.1002197265625, 2.712890625, 0.50341796875, 0.9189453125, -0.53125, -0.64990234375, 0.6875, 0.268310546875, -0.4140625, -0.51953125, -2.16796875, -0.05780029296875, 1.3251953125, -0.95458984375, 0.107177734375, 1.431640625, 0.9921875, -0.60302734375, -2.205078125, 1.6435546875, 0.227783203125, -0.189697265625, 1.12890625, 1.1640625, 0.102783203125, 0.0088958740234375, -0.43115234375, 0.84521484375, -2.232421875, -0.2296142578125, 0.49462890625, -0.27587890625, 0.88525390625, 1.669921875, -0.421142578125, 1.154296875, -0.09539794921875, 0.0058441162109375, 1.8525390625, -0.2481689453125, 0.28955078125, 1.0712890625, 0.85888671875, 0.2484130859375, 1.4140625, -1.7880859375, -0.634765625, 0.53564453125, -2.4609375, -0.1419677734375, 0.238525390625, -0.368408203125, 0.490234375, 1.6640625, 0.7431640625, -0.470947265625, -1.154296875, -0.53759765625, -1.056640625, 0.392822265625, -0.02630615234375, 0.5087890625, 0.51220703125, -0.6572265625, 0.19775390625, 0.91259765625, -0.62158203125, 0.7763671875, -0.966796875, 1.2744140625, 0.469482421875, -0.339599609375, 0.18505859375, -0.51953125, -1.6708984375, -2.427734375, -1.1533203125, 0.8037109375, 0.87548828125, -2.12890625, -0.17138671875, -0.07733154296875, 1.4140625, 1.3955078125, -1.419921875, -1.095703125, 1.0390625, 0.5673828125, -0.712890625, -0.9833984375, 2.068359375, -0.306396484375, -0.422119140625, 0.469482421875, 0.244873046875, -0.0799560546875, 0.01123809814453125, 0.54052734375, 0.302490234375, 2.091796875, 0.39013671875, 2.3984375, 1.4765625, 0.215576171875, 0.71044921875, -0.12188720703125, 0.7236328125, -0.4765625, -0.98779296875, 1.25390625, -0.2498779296875, -0.47509765625, -0.74609375, 0.3525390625, -0.57763671875, -0.10760498046875, 0.218017578125, -0.70361328125, 0.271240234375, -0.783203125, 0.10205078125, 0.11883544921875, -0.51171875, 0.921875, -0.046356201171875, -0.46875, -0.6015625, 0.322021484375, -1.1474609375, -0.178955078125, -0.64501953125, 1.8447265625, -0.29638671875, 0.376953125, 0.035797119140625, 0.517578125, -1.2744140625, 1.5126953125, -0.79638671875, -1.19921875, 0.50048828125, 0.030029296875, -1.4345703125, 1.5146484375, -1.4375, -1.4169921875, 0.8876953125, 0.70947265625, -1.0263671875, 0.426025390625, 0.0269012451171875, 0.2398681640625, -1.638671875, 0.28662109375, -2.201171875, -1.115234375, 1.6396484375, -0.07171630859375, -0.438720703125, 0.53564453125, -1.6943359375, 0.87646484375, 0.80322265625, -1.4091796875, -0.0870361328125, 1.662109375, -1.478515625, -1.3046875, 0.7685546875, -0.9326171875, -0.3037109375, 0.492431640625, 0.91357421875, 0.33154296875, 0.1573486328125, 0.367919921875, -1.46875, -0.05108642578125, -0.849609375, -1.1767578125, 0.0755615234375, -2.173828125, 1.98046875, -1.375, -1.134765625, -0.98876953125, 1.07421875, -0.153564453125, 0.41796875, -0.98974609375, -2.37890625, -0.228515625, -2.451171875, -0.1279296875, 0.28466796875, -1.0546875, -0.666015625, -1.560546875, 1.23828125, 0.69970703125, 1.2109375, 0.7021484375, 0.8994140625, -0.1805419921875, -0.55322265625, 0.0970458984375, -0.69189453125, -0.055908203125, -0.474365234375, 0.2353515625, -0.38427734375, -0.4345703125, -0.27392578125, -0.91015625, -0.08612060546875, -0.76708984375, 0.60595703125, 0.304931640625, 1.7197265625, 1.7509765625, -0.94921875, 0.794921875, 1.677734375, 0.190673828125, -0.26025390625, 0.66796875, -1.189453125, 0.403076171875, 1.8984375, -0.578125, -0.425537109375, 0.74365234375, -0.046661376953125, 0.52392578125, -0.11065673828125, -0.96337890625, -0.1380615234375, 0.18505859375, 1.8037109375, -1.3935546875, -2.0078125, -0.7041015625, -1.462890625, -1.515625, 0.8369140625, -1.2607421875, -0.2471923828125, 0.300048828125, 0.196044921875, 1.7119140625, 1.1494140625, -0.375244140625, -0.317626953125, 0.1546630859375, -0.63525390625, 1.8017578125, -1.0400390625, -1.0810546875, -0.853515625, -0.06500244140625, -1.09765625, 0.155029296875, 0.7490234375, -1.3046875, -0.69970703125, -1.1533203125, -0.72412109375, -0.09075927734375, -1.0517578125, 0.1595458984375, -1.0419921875, 0.52880859375, 0.84423828125, -1.0830078125, 0.34375, 0.06512451171875, -1.1416015625, 1.3017578125, 0.1041259765625, -0.59716796875, 0.423095703125, -0.61572265625, 0.37744140625, 0.10943603515625, 2.025390625, -0.46728515625, -0.73974609375, 0.288330078125, -1.1513671875, -0.50048828125, -3.107421875, -0.6484375, -1.013671875, 0.072265625, -0.3408203125, -0.2266845703125, 1.763671875, 0.142333984375, -1.6162109375, 0.3837890625, -0.5625, -0.279052734375, -0.77001953125, -1.896484375, -1.375, 0.000492095947265625, -1.0673828125, 1.314453125, 1.21484375, 0.12060546875, -0.5029296875, 2.09375, 0.1651611328125, -0.273681640625, -0.4228515625, 1.0888671875, 0.0323486328125, -0.02099609375, -0.71728515625, 0.42333984375, -0.128173828125, 1.11328125, 0.1002197265625, 0.62158203125, -0.116943359375, -1.4140625, -1.216796875, -0.60791015625, -0.90771484375, -0.317138671875, 0.5966796875, -0.2064208984375, -0.74560546875, 1.359375, 0.70068359375, 0.50439453125, 0.241943359375, 0.41015625, 2.61328125, -0.87646484375, 1.64453125, 0.1876220703125, -0.7236328125, -0.06707763671875, 0.447509765625, 0.64453125, 0.412841796875, 1.07421875, -0.1942138671875, 0.60302734375, 0.2587890625, -0.7685546875, 1.392578125, 0.307373046875, 0.48681640625, -0.318359375, -0.467529296875, 1.404296875, -1.0400390625, -0.12139892578125, -0.865234375, 1.3427734375, -1.8779296875, 1.173828125, -0.9931640625, -0.4560546875, 0.1864013671875, 0.4794921875, -0.317138671875, 2.0078125, 0.44970703125, 0.1810302734375, 0.33203125, 0.382080078125, -0.003360748291015625, -0.443359375, -0.5126953125, -0.0616455078125, -0.40234375, -0.05908203125, 1.865234375, -1.244140625, 0.8056640625, -1.39453125, 0.47900390625, -0.646484375, 1.1787109375, 0.97998046875, 0.89208984375, 1.017578125, -0.6904296875, 0.467041015625, 1.25, -1.466796875, -0.0161895751953125, 0.6513671875, 1.5947265625, -1.845703125, 0.42236328125, -1.033203125, -0.176025390625, 0.5546875, -0.73388671875, -0.493408203125, 1.4091796875, 1.2626953125, -0.76318359375, -0.65625, 2.49609375, -0.206787109375, -0.09222412109375, -0.489501953125, -0.33349609375, -0.859375, -0.94091796875, 0.04736328125, -2.5390625, -0.376708984375, -1.2158203125, 0.036163330078125, 2.216796875, -0.1708984375, 0.2177734375, -0.693359375, 0.20654296875, -0.1300048828125, 0.89111328125, 0.9609375, -1.7509765625, -1.0615234375, -1.224609375, 1.2724609375, 0.63671875, -1.125, -0.320068359375, -0.67041015625, -1.1748046875, -0.1806640625, -0.07855224609375, -1.2607421875, 2.17578125, -0.0096282958984375, 0.53662109375, -0.1390380859375, -0.284912109375, -0.393310546875, 0.27294921875, -0.264892578125, -0.07745361328125, -0.369384765625, 0.05804443359375, -0.17626953125, -0.63134765625, -1.6259765625, 1.314453125, 1.2705078125, 0.0103302001953125, -1.46875, -1.900390625, 2.654296875, 0.07867431640625, 1.9423828125, 0.74853515625, 0.07684326171875, -0.2344970703125, -1.77734375, -0.2052001953125, -0.6630859375, -0.6220703125, -1.994140625, -0.309326171875, 0.322998046875, -0.86767578125, 1.8408203125, -0.1553955078125, -0.71044921875, -2.04296875, 0.88671875, -0.681640625, 0.4599609375, 1.060546875, -0.7548828125, 1.0126953125, 0.2353515625, 0.96337890625, 0.0931396484375, -0.225341796875, -1.9189453125, -0.14990234375, 0.79052734375, 0.281494140625, 0.06390380859375, 0.404541015625, 0.476318359375, 0.370361328125, 0.02880859375, -0.65625, -0.24365234375, -0.385009765625, 0.289306640625, 0.034088134765625, -0.319580078125, -0.55419921875, 2.3359375, 0.8544921875, -1.677734375, 1.87109375, 0.371826171875, 0.8603515625, -0.2301025390625, 0.51806640625, -0.900390625, -0.41455078125, 1.53515625, -0.1796875, -0.94140625, -1.009765625, -0.069580078125, -0.208740234375, 0.82958984375, 0.51171875, -0.326904296875, -0.1080322265625, -0.460205078125, 0.0537109375, 1.9833984375, -0.465576171875, 0.0185546875, -0.736328125, 1.1943359375, -1.5341796875, -0.51611328125, -1.6875, -0.68115234375, 0.17626953125, 1.392578125, -0.2418212890625, 1.126953125, 0.347900390625, 0.64111328125, 1.849609375, 0.34033203125, -0.11248779296875, -0.8701171875, 0.00971221923828125, 1.46875, 1.6494140625, 0.2210693359375, 0.7548828125, 0.393798828125, -0.408203125, 0.6904296875, -0.76806640625, 0.5830078125, 0.251953125, -0.88525390625, 1.6630859375, 1.443359375, -1.236328125, -0.36767578125, 0.1190185546875, 0.810546875, 0.465576171875, -0.1534423828125, 0.75439453125, 0.01323699951171875, -0.106689453125, 1.26953125, -0.117431640625, -0.50439453125, -1.8486328125, -1.369140625, 0.9248046875, 0.278564453125, 0.025238037109375, -0.417236328125, -1.8046875, 0.701171875, -2.138671875, -1.0517578125, -0.462158203125, 0.62890625, 1.0361328125, 1.10546875, -0.59130859375, 0.513671875, 2.33203125, -1.080078125, -1.7041015625, -0.2578125, 0.4267578125, 0.392578125, 0.982421875, 0.9892578125, -1.0458984375, -1.580078125, -0.50927734375, 0.20361328125, -1.3251953125, 1.962890625, -0.54736328125, -0.37060546875, -2.84765625, 1.26953125, -0.2108154296875, 0.1890869140625, 1.9560546875, -0.6611328125, -1.6708984375, 1.900390625, -1.0595703125, 1.25390625, -2.107421875, 1.69140625, -0.057708740234375, 0.2861328125, 0.97119140625, 0.54833984375, 0.83203125, -0.7490234375, 1.5009765625, -0.343994140625, -0.096923828125, -1.3642578125, 1.5322265625, 1.171875, -0.10882568359375, -0.26513671875, -0.76513671875, 0.2464599609375, 1.8525390625, 1.9892578125, 0.7763671875, 0.7587890625, 0.2607421875, -0.53564453125, 1.0537109375, -0.307861328125, -2.01953125, 0.470947265625, 1.2822265625, -0.71337890625, 0.365478515625, -0.56396484375, 0.212890625, 1.5078125, 1.7978515625, 0.336181640625, -0.2099609375, 0.56005859375, -1.044921875, 0.62548828125, -0.44873046875, 1.2197265625, -0.46142578125, -0.44873046875, -0.50146484375, -0.412353515625, -1.138671875, 0.431640625, -0.2027587890625, -0.8125, -2.04296875, -0.97314453125, -0.67431640625, -0.323486328125, -0.1683349609375, -0.03125, 0.248291015625, 0.80419921875, -0.65966796875, 0.64404296875, -0.1412353515625, 0.8974609375, 1.625, 0.57275390625, 0.833984375, -0.031982421875, -1.1533203125, -0.290771484375, -0.1119384765625, 0.280029296875, 0.054901123046875, -0.26025390625, -0.24853515625, -0.031463623046875, 1.1904296875, -0.94921875, -0.159423828125, -0.1937255859375, -1.896484375, -0.0430908203125, 0.298583984375, 2.177734375, 0.87353515625, 1.8076171875, -0.4609375, 0.619140625, -1.6943359375, -0.611328125, 1.638671875, 0.62158203125, 0.310546875, -0.3154296875, -0.5732421875, -1.310546875, -0.2152099609375, -2.103515625, 0.313720703125, -0.7216796875, 0.80615234375, -1.18359375, -1.4658203125, 0.07647705078125, 0.354248046875, 1.6865234375, 0.042572021484375, -0.50244140625, 0.07012939453125, 1.4140625, 1.4921875, -1.173828125, -1.087890625, 0.67578125, -0.89453125, 0.91455078125, -1.1923828125, 0.2110595703125, 0.5615234375, -0.264404296875, -0.043731689453125, -0.1468505859375, -0.51953125, 0.87939453125, -0.06085205078125, -0.62158203125, -0.264404296875, -0.3076171875, 0.0012578964233398438, -1.7646484375, 1.2705078125, -0.65478515625, 0.7158203125, 0.53271484375, -1.248046875, -0.69921875, -0.08941650390625, -0.3740234375, 0.501953125, -0.357666015625, -0.0810546875, 0.274169921875, -0.061981201171875, -1.552734375, 0.2132568359375, -0.46142578125, 0.5205078125, 1.0078125, 0.73388671875, 0.41162109375, -1.490234375, -0.291259765625, 2.98046875, -0.86181640625, -1.3486328125, -0.548828125, 2.560546875, 1.0537109375, -0.10479736328125, 0.08746337890625, -1.3955078125, 1.3173828125, -2.181640625, 0.019287109375, 1.0634765625, -1.6640625, 0.126708984375, 1.412109375, -0.5673828125, 0.373291015625, -1.390625, -0.36376953125, -0.2919921875, 0.2259521484375, -1.103515625, -0.99658203125, -0.00811004638671875, 0.283935546875, 0.32763671875, -1.146484375, 0.81494140625, -0.339111328125, -0.78271484375, 1.541015625, 0.77001953125, 1.46484375, -0.365478515625, -0.1610107421875, -0.43115234375, -1.4208984375, 2.0, 0.26953125, -0.76171875, 2.052734375, -1.298828125, -0.8232421875, 0.4853515625, -2.4765625, -0.703125, 0.9111328125, -1.07421875, -1.2041015625, -0.57470703125, -1.8837890625, 0.58740234375, 0.8271484375, 2.494140625, 0.541015625, 0.467529296875, 0.64013671875, -0.48291015625, -0.436767578125, 0.87255859375, 0.3857421875, -0.853515625, -1.265625, 0.372802734375, -1.009765625, 1.166015625, -1.2001953125, -1.490234375, 1.7744140625, 0.46484375, 1.486328125, -0.1934814453125, -0.319091796875, 1.146484375, -0.2149658203125, 0.177734375, -0.29345703125, 1.1533203125, -0.406005859375, -1.6181640625, 0.98779296875, -0.32421875, 1.0302734375, -0.426025390625, 0.70751953125, -1.2216796875, -0.251708984375, -1.2783203125, -1.1162109375, -0.0216522216796875, 0.0345458984375, -0.396484375, 0.7001953125, -0.72216796875, -2.50390625, 0.322021484375, -0.1297607421875, -1.1962890625, -1.6953125, -1.11328125, 0.29052734375, -0.84130859375, 0.19873046875, -0.327880859375, -0.37841796875, -0.0626220703125, 0.9033203125, -1.0009765625, 2.345703125, 0.363037109375, 0.56884765625, 0.153076171875, -2.361328125, 0.7685546875, 0.45703125, 1.3134765625, -0.56787109375, 1.5322265625, 0.0718994140625, 0.70068359375, -0.173583984375, 1.8876953125, -0.451171875, 0.84619140625, 0.13525390625, 0.3935546875, 1.37109375, 0.93994140625, 0.65966796875, -0.57958984375, -2.427734375, 1.9501953125, 1.3076171875, 1.3623046875, -0.94140625, -0.185302734375, -0.7802734375, 2.001953125, -0.55078125, -1.4091796875, 0.08294677734375, 2.173828125, 1.484375, 1.76171875, 0.1893310546875, 1.0703125, -1.1328125, 0.0164031982421875, -0.76953125, 0.367431640625, 1.6640625, 0.053009033203125, -2.796875, 0.10516357421875, 0.73779296875, 1.7900390625, -1.9453125, 0.114013671875, 0.42626953125, 0.457763671875, -0.66162109375, 0.5771484375, 0.33447265625, -1.287109375, 0.1131591796875, 0.0144500732421875, -0.130615234375, 0.50048828125, -1.4873046875, -1.4443359375, -0.075439453125, 0.58447265625, -0.016571044921875, -1.939453125, -1.173828125, -0.1412353515625, 0.2340087890625, -1.513671875, -0.818359375, -0.138427734375, -0.30126953125, -0.58935546875, -0.3388671875, 0.50830078125, -0.284423828125, 0.771484375, -0.09124755859375, 0.394287109375, 0.970703125, 0.97021484375, -1.0419921875, 0.8642578125, 0.9970703125, -0.049163818359375, 0.62939453125, -0.157470703125, -0.63818359375, -1.0263671875, 0.25927734375, 0.6826171875, -0.95068359375, 0.01049041748046875, -0.72802734375, 0.92822265625, -0.8662109375, -0.67041015625, -0.43701171875, -0.062103271484375, -1.8447265625, -0.79052734375, 0.44091796875, 0.0509033203125, -0.64306640625, 0.94482421875, 1.14453125, -0.034088134765625, -0.26904296875, -0.003787994384765625, 0.8291015625, -0.09844970703125, 1.2744140625, 1.7607421875, 0.97705078125, 0.16064453125, -0.833984375, 1.11328125, 0.8359375, 1.7666015625, 1.5068359375, -1.203125, -0.71044921875, -0.59033203125, 1.2109375, 0.0782470703125, -0.76416015625, -0.0938720703125, -0.369384765625, 0.52587890625, 2.759765625, -0.6259765625, 1.1787109375, -0.357177734375, 0.65869140625, 1.0869140625, -0.541015625, 0.61083984375, 1.203125, -1.5009765625, -0.51318359375, -0.78125, -1.361328125, 0.192626953125, 1.2900390625, 1.01953125, 0.458984375, -0.329833984375, 1.5126953125, -1.9189453125, 1.4384765625, -0.7080078125, 0.075927734375, -0.142822265625, 1.2158203125, 1.1484375, -1.111328125, 1.9091796875, -0.82080078125, -0.07867431640625, -0.040191650390625, -2.044921875, -0.765625, 0.065673828125, 2.546875, 0.039031982421875, -1.224609375, 0.6962890625, -1.12890625, 0.362060546875, 1.279296875, 0.155517578125, 1.0771484375, -0.336181640625, 0.447998046875, 0.19287109375, 0.470947265625, -0.0222930908203125, -0.6337890625, 0.5361328125, -0.265380859375, -1.9541015625, 0.021392822265625, 1.0654296875, 0.57470703125, 0.433349609375, 0.66455078125, 0.12152099609375, -0.146240234375, -1.4306640625, -0.77783203125, -1.6806640625, -0.086669921875, -1.2763671875, 0.06005859375, 2.142578125, 0.201171875, 0.251953125, -1.0791015625, -1.173828125, 0.826171875, -1.310546875, 0.62841796875, 1.85546875, 0.10577392578125, 0.8779296875, -0.86572265625, -1.189453125, -0.8798828125, 0.477783203125, 0.845703125, -0.4892578125, 0.10552978515625, -0.0379638671875, -0.127685546875, -1.2470703125, 1.2314453125, 0.61376953125, 0.26220703125, -0.9111328125, -0.296875, -1.232421875, 1.2265625, -0.5869140625, 1.068359375, -1.171875, -0.30859375, -0.402099609375, 2.05859375, 0.77734375, 0.187744140625, -0.4697265625, -1.1884765625, 2.416015625, -0.472900390625, 1.138671875, -1.328125, 0.01279449462890625, 0.8134765625, 0.04449462890625, -1.0859375, -1.232421875, 0.23486328125, 0.68017578125, -0.4716796875, 0.282470703125, -0.039154052734375, -0.137939453125, -0.01039886474609375, -1.390625, 0.315185546875, 1.95703125, 0.216552734375, -0.182861328125, 0.26416015625, -0.47021484375, 0.95556640625, 0.78466796875, 0.92333984375, 1.87890625, 1.4912109375, 0.26416015625, -1.08984375, -0.9072265625, 0.26123046875, 0.6845703125, -1.3095703125, -0.90283203125, 0.6552734375, -0.287109375, 0.0531005859375, -0.1317138671875, 1.7265625, 0.85595703125, -0.7666015625, -0.0791015625, 0.66650390625, -0.352783203125, 0.24609375, 0.19580078125, -0.148193359375, 1.5986328125, 0.13134765625, -0.49072265625, -0.64599609375, -0.7587890625, -1.5458984375, -0.6318359375, -0.1568603515625, 0.68115234375, 0.93896484375, 0.5439453125, -0.2467041015625, -0.3271484375, 0.94384765625, 2.43359375, 1.6171875, -1.36328125, 2.16015625, -0.53076171875, -0.10552978515625, 1.4482421875, 0.76416015625, 1.708984375, 0.87451171875, 0.204833984375, 1.3125, 1.3134765625, 0.2421875, 0.55419921875, 0.080322265625, -1.0498046875, -0.034698486328125, -1.5498046875, -0.2347412109375, -0.779296875, 0.91943359375, -0.8837890625, 0.1971435546875, -1.546875, 0.1907958984375, 1.1494140625, -0.82763671875, 0.288330078125, 1.3994140625, -0.61669921875, 0.2724609375, 0.9091796875, 1.1279296875, -0.32080078125, 1.5673828125, -0.70751953125, -0.74951171875, -0.7060546875, -0.07830810546875, 0.25, 2.306640625, -0.0841064453125, -1.1689453125, 0.10455322265625, -1.8701171875, 0.89794921875, -0.340087890625, 0.78564453125, 0.0081787109375, -0.428955078125, -0.0096282958984375, 0.615234375, 0.14697265625, -0.79443359375, -0.7998046875, -2.0546875, -1.22265625, 0.489990234375, 1.5634765625, 0.439453125, -0.6796875, 0.80517578125, -1.7998046875, -0.1365966796875, -0.44775390625, -0.33447265625, 0.0143585205078125, 0.7861328125, 0.4765625, -1.4130859375, -2.236328125, 1.52734375, 1.3916015625, -1.181640625, 1.3193359375, 0.703125, -0.349365234375, 0.465576171875, -0.56787109375, -0.7236328125, 0.95458984375, -0.77099609375, -0.237060546875, -0.61474609375, -0.284423828125, -3.3203125, -0.419677734375, 0.1549072265625, 0.51904296875, 0.47216796875, -0.1766357421875, -0.37890625, 0.2099609375, -0.43212890625, 0.352783203125, -2.080078125, 0.83056640625, -0.406005859375, -0.423583984375, 0.8369140625, -1.9296875, 0.57470703125, 0.99609375, 0.4755859375, -0.6376953125, -0.1995849609375, -1.2685546875, 0.6396484375, -0.022796630859375, 0.84228515625, 0.1593017578125, 0.3671875, -1.1416015625, -0.2391357421875, -0.229248046875, 0.181396484375, -0.260498046875, -0.6630859375, -1.6962890625, 0.22314453125, -1.0966796875, -0.12371826171875, 1.6337890625, -1.001953125, -0.75439453125, -1.541015625, 0.625, -0.049407958984375, -1.025390625, -0.418212890625, -0.1839599609375, -0.69189453125, -1.0029296875, -1.0205078125, -0.943359375, 0.7587890625, 1.41796875, 2.150390625, -1.287109375, -1.3173828125, -1.259765625, 0.55517578125, -0.211181640625, -1.1435546875, -0.68359375, -0.11541748046875, -1.0947265625, -1.2216796875, -0.501953125, 2.296875, 0.03875732421875, 0.009979248046875, 0.57958984375, 0.3173828125, -0.202392578125, -0.388916015625, 0.94140625, 0.224609375, -0.169921875, -0.978515625, 0.21923828125, -2.3046875, -1.1708984375, 0.92138671875, 0.0169677734375, -0.026885986328125, -0.0908203125, 0.442138671875, -0.34326171875, 1.5673828125, -0.470703125, 0.61083984375, -0.319091796875, -0.220947265625, 0.02789306640625, 1.1279296875, -0.36376953125, -0.57275390625, -0.361572265625, 0.7275390625, 2.34375, -1.861328125, 0.336181640625, -0.84716796875, -0.98193359375, 0.233642578125, 0.251220703125, -0.8310546875, 0.73388671875, -0.69384765625, 0.89697265625, -0.78466796875, -1.1962890625, 1.51171875, 0.28271484375, 0.9580078125, 1.2431640625, 0.441162109375, -0.085693359375, -0.17236328125, -0.9951171875, -2.849609375, -1.2431640625, -0.291015625, 0.19873046875, 0.2939453125, -0.67724609375, 0.54150390625, 0.80078125, -0.978515625, 0.4345703125, -0.90625, -0.78564453125, 0.177734375, -0.6162109375, 0.8251953125, -1.4912109375, 0.2237548828125, 0.36669921875, -0.6533203125, 0.892578125, -0.0777587890625, -1.2080078125, -0.96728515625, 0.5185546875, 1.111328125, -0.80517578125, 1.3408203125, -0.91064453125, -0.04547119140625, 0.794921875, 0.65673828125, -0.242431640625, -0.65087890625, -0.7705078125, 0.427734375, 0.2366943359375, 0.58544921875, -0.1373291015625, 0.54345703125, 0.0643310546875, -0.4404296875, -0.0135650634765625, 0.16552734375, -1.283203125, -0.277587890625, 0.446044921875, -1.041015625, -0.9931640625, 1.2109375, 1.158203125, -0.30859375, -0.31103515625, 1.9794921875, 0.8388671875, 1.162109375, 0.8662109375, -0.67724609375, 0.294189453125, 0.7314453125, -0.52978515625, -0.67822265625, -0.71923828125, 0.4453125, 1.1982421875, 2.61328125, -0.71484375, -0.42236328125, -0.275390625, 0.38525390625, 1.0048828125, 1.2041015625, -0.5927734375, -0.3349609375, 0.42041015625, 0.443359375, -0.4853515625, 0.8486328125, 0.4453125, 1.0263671875, -1.3408203125, 0.90576171875, -0.861328125, -0.59375, -0.2548828125, 0.400634765625, -0.405029296875, -0.1650390625, -0.48291015625, -0.921875, 1.19140625, -0.78662109375, 0.830078125, 0.58837890625, 0.34765625, -1.28515625, -6.216764450073242e-05, 0.309326171875, -2.345703125, 1.2431640625, -0.40283203125, 0.480712890625, -0.303955078125, -0.5625, -0.371826171875, -1.3818359375, -1.66796875, -0.422607421875, -0.476806640625, -0.101318359375, 0.74365234375, 0.350341796875, -0.0008444786071777344, 0.505859375, -1.2431640625, 0.1407470703125, -0.6826171875, 1.958984375, 0.09368896484375, 0.36376953125, 0.1834716796875, 0.129150390625, -0.9638671875, 1.7509765625, -0.44921875, 0.441162109375, 1.6650390625, -0.0065155029296875, 0.71826171875, -0.43310546875, -2.068359375, -0.6435546875, 0.5625, 1.4814453125, 0.034759521484375, 0.1669921875, 0.1533203125, -0.289306640625, 0.2197265625, 0.537109375, 0.27001953125, -0.1783447265625, 0.490478515625, -2.126953125, -0.2403564453125, -1.109375, -1.3525390625, -0.1307373046875, 0.62451171875, -1.517578125, 0.286865234375, 0.019866943359375, -1.08984375, -1.318359375, 0.270751953125, -2.154296875, 0.05621337890625, -0.376220703125, 1.0224609375, 0.34326171875, -0.39599609375, -0.7763671875, -1.2109375, 0.8203125, 0.157470703125, 1.0595703125, -1.7724609375, 1.025390625, -0.65283203125, -1.5673828125, -0.451416015625, -0.04437255859375, 0.089111328125, 1.90234375, 0.59765625, -0.1356201171875, -1.3310546875, 0.045745849609375, -0.91650390625, -0.916015625, -0.48095703125, -1.7705078125, -0.69287109375, 1.4892578125, -0.072509765625, 0.53564453125, -0.81298828125, -0.6513671875, 1.38671875, -1.146484375, 0.8173828125, 0.65771484375, -0.10791015625, 0.158203125, -0.9306640625, 1.3046875, -1.2998046875, -0.64013671875, -0.054412841796875, 0.66845703125, 1.4541015625, -0.67626953125, -0.315673828125, -0.77099609375, -0.2010498046875, 1.5068359375, -0.84375, -2.744140625, 2.33984375, -1.185546875, -2.1875, -1.1171875, 0.66943359375, 0.810546875, 0.685546875, -1.1689453125, 0.27685546875, 0.421630859375, -0.421142578125, 1.36328125, 1.9755859375, -0.4228515625, -2.005859375, 0.275634765625, -0.1260986328125, -0.72021484375, 0.86474609375, 0.01454925537109375, 0.5849609375, 0.3515625, -1.0859375, 0.282470703125, -0.299072265625, -0.65869140625, 0.3642578125, -0.6162109375, -0.43359375, -1.03125, 1.5927734375, -0.09783935546875, 1.9111328125, -0.84033203125, 2.30859375, 0.311767578125, 0.74658203125, -0.85498046875, -0.52197265625, 0.849609375, -0.8623046875, -0.0953369140625, -0.8427734375, -0.430419921875, -1.201171875, 0.0293731689453125, 1.0712890625, -0.26708984375, -2.0546875, 0.51953125, 0.658203125, -0.70703125, 0.841796875, 1.2451171875, 0.23193359375, 0.142578125, -0.333251953125, -0.0128936767578125, -0.6728515625, -0.95263671875, -1.01953125, -0.9951171875, -0.6201171875, -1.3017578125, -1.83203125, -0.299560546875, 0.615234375, -0.21630859375, 0.517578125, -0.57177734375, -0.1387939453125, -0.05523681640625, 0.270751953125, -0.9541015625, 0.235107421875, 0.04937744140625, -0.495361328125, -0.595703125, 1.865234375, -1.3916015625, -1.0546875, -0.77294921875, 0.70654296875, -0.99609375, -1.513671875, 0.9736328125, 0.59619140625, -0.391357421875, 0.54736328125, -0.62939453125, 0.50048828125, -0.1297607421875, -1.484375, 0.0369873046875, 1.3173828125, 1.630859375, -0.44775390625, 1.724609375, 0.99755859375, 1.1396484375, -0.53466796875, 0.79541015625, -0.432373046875, 0.2154541015625, -0.46240234375, 1.814453125, -0.453369140625, -0.45068359375, 1.333984375, 0.56884765625, -0.3203125, -0.224365234375, 0.0343017578125, 1.283203125, 1.20703125, 0.331298828125, -1.1240234375, -0.28759765625, -0.599609375, -0.607421875, 1.2705078125, 1.7880859375, -0.2254638671875, 0.82763671875, -1.4755859375, 0.61572265625, -0.287109375, -0.66015625, 0.460205078125, 0.41552734375, 1.697265625, 0.342529296875, -0.96826171875, -1.4775390625, 0.1920166015625, -0.004756927490234375, -0.72802734375, 0.70458984375, 0.03857421875, -0.9111328125, -0.1468505859375, 0.478759765625, 1.0068359375, -0.23974609375, -0.71728515625, -0.285888671875, -0.615234375, 1.291015625, -1.6796875, 0.76416015625, -0.1722412109375, 0.41552734375, -1.20703125, 0.708984375, -0.276611328125, -1.21875, 0.91015625, 0.49609375, 0.288818359375, -1.7880859375, -1.0634765625, -1.275390625, -2.16015625, 0.59375, 0.4658203125, -2.396484375, -1.3779296875, 0.26171875, -0.192138671875, 0.962890625, 0.1336669921875, -1.01171875, -0.84716796875, 0.1444091796875, 0.143310546875, -0.595703125, 1.44140625, 0.533203125, -1.884765625, -1.0654296875, 2.3671875, 1.1484375, 0.099365234375, 0.5537109375, 2.75, 0.3232421875, -1.3017578125, 0.346435546875, -1.873046875, -0.57373046875, -0.486572265625, 0.072021484375, 1.484375, -0.56005859375, -0.78125, 0.2364501953125, -0.7568359375, -1.3984375, -1.322265625, -0.42431640625, -0.7705078125, 0.37255859375, 0.459228515625, 0.50537109375, -0.33984375, 0.943359375, 0.321044921875, 0.88232421875, -0.49951171875, 0.8388671875, 0.3935546875, -0.77294921875, 0.287353515625, -0.29248046875, -0.231689453125, 1.6259765625, -0.11016845703125, 0.986328125, -0.962890625, -2.279296875, -0.57861328125, 1.2568359375, -0.7744140625, -1.53125, -0.91064453125, -0.389404296875, -0.316162109375, 1.3828125, -0.1884765625, -0.477294921875, -0.06842041015625, -0.6796875, -1.6376953125, -0.48486328125, 0.3955078125, -1.220703125, 1.3271484375, 0.5810546875, -1.3583984375, -0.00997161865234375, -0.44677734375, 2.0703125, 0.2080078125, 0.1781005859375, 0.26806640625, 0.69384765625, 0.0753173828125, -1.068359375, -0.60400390625, -0.8349609375, 1.56640625, -0.7236328125, -0.6650390625, 0.5576171875, 0.0080718994140625, -0.6005859375, 1.3857421875, -0.69287109375, -2.154296875, 0.53125, -0.78662109375, -1.447265625, 0.36181640625, 0.52783203125, -0.1353759765625, -0.55517578125, -1.1162109375, -1.244140625, -0.0047607421875, -0.266845703125, 1.3876953125, -1.505859375, 1.212890625, 0.1304931640625, -2.455078125, 1.0771484375, 0.270751953125, -0.475341796875, 0.732421875, -0.11328125, -0.330322265625, 1.9013671875, -1.7685546875, 1.3173828125, -0.128662109375, 2.083984375, 0.0963134765625, 2.857421875, -1.2333984375, -0.72216796875, 0.03753662109375, -0.59521484375, -0.365478515625, -0.5498046875, 1.5927734375, 0.3193359375, -0.537109375, -0.85986328125, -0.58984375, -2.453125, 1.1728515625, 1.5634765625, -1.748046875, -1.1328125, 1.5439453125, -0.53369140625, 0.86181640625, 0.5849609375, 0.8076171875, 1.7919921875, -0.7314453125, 0.7529296875, -0.60205078125, -0.16796875, -0.348876953125, -0.681640625, -0.68310546875, 0.8154296875, 0.044525146484375, 0.1004638671875, 1.1181640625, 0.83154296875, 0.1036376953125, 0.71630859375, -1.623046875, 2.86328125, 0.2252197265625, 0.64501953125, 1.0634765625, 0.10931396484375, -0.8935546875, -0.96923828125, -0.11181640625, -1.4130859375, 1.322265625, -0.1065673828125, -1.375, -0.145751953125, -0.77880859375, -0.62109375, -1.9072265625, 1.5478515625, -0.2294921875, -2.46875, -0.090087890625, 0.93310546875, 1.08984375, -0.352783203125, 0.09185791015625, -0.6904296875, 0.4287109375, -0.44921875, -0.50830078125, -1.25, -0.371337890625, -0.162109375, 0.50341796875, 0.418701171875, -0.08026123046875, -0.47314453125, 0.99072265625, 2.1953125, 0.486328125, -1.51171875, -1.244140625, 0.143798828125, -0.1033935546875, -1.3671875, -1.0341796875, -1.578125, 0.86669921875, -0.03570556640625, 0.50439453125, -0.347900390625, -0.55810546875, -0.7939453125, 0.2327880859375, 1.322265625, 0.25390625, 1.060546875, 0.5419921875, 0.021087646484375, 0.48583984375, -0.9677734375, 0.849609375, -0.347412109375, 0.763671875, 1.091796875, 0.392578125, -1.212890625, -0.036529541015625, -0.70361328125, 0.82080078125, -0.46923828125, 0.5791015625, -0.128173828125, 0.035858154296875, -0.204345703125, 0.403564453125, -1.14453125, 1.0146484375, 0.638671875, 1.5517578125, 0.6298828125, -1.0419921875, -1.7890625, -0.5087890625, 0.7646484375, 0.47705078125, 0.72802734375, -0.896484375, 0.328125, -0.2548828125, 0.35595703125, 1.439453125, -0.263427734375, 1.5107421875, 1.1103515625, 0.8076171875, 0.2401123046875, -1.0751953125, 0.489501953125, -1.583984375, -0.46484375, -2.296875, 0.93798828125, 0.15966796875, -0.209228515625, 1.0126953125, -1.146484375, -1.5419921875, -2.232421875, -1.046875, -1.5302734375, -1.5048828125, 1.5478515625, -0.058441162109375, 1.5048828125, 0.9208984375, -0.238037109375, 0.306884765625, -1.12109375, -2.0625, -0.1572265625, -0.650390625, 0.237060546875, -0.485595703125, -0.9375, 0.03167724609375, -1.0625, -0.5634765625, 1.150390625, 0.267822265625, 0.61962890625, -0.385986328125, -1.822265625, -0.03607177734375, -1.6240234375, 0.2008056640625, 1.00390625, -2.70703125, -0.059600830078125, -1.35546875, -0.1878662109375, 0.380859375, 0.25048828125, 0.7490234375, -1.630859375, 0.004764556884765625, 2.095703125, -1.1572265625, 1.955078125, 0.80224609375, -0.2310791015625, 1.587890625, 0.238037109375, -1.7724609375, -0.225341796875, 1.919921875, 1.5283203125, -0.12408447265625, 0.716796875, 0.1300048828125, -1.7978515625, 0.2978515625, -1.009765625, -2.541015625, 1.3583984375, -2.40234375, 0.224609375, -0.443603515625, -0.4765625, 0.59765625, -0.833984375, -0.69384765625, 0.98974609375, 0.10870361328125, -0.040985107421875, -0.4287109375, -0.31591796875, 0.2425537109375, -0.93310546875, 0.5966796875, -1.5771484375, 0.912109375, 0.8134765625, -0.31201171875, -0.43310546875, 1.33984375, -0.17822265625, 0.0556640625, 1.5322265625, 0.73193359375, 0.8154296875, -1.3662109375, 0.611328125, -0.26611328125, 0.0116119384765625, -0.280029296875, 1.1162109375, -0.470947265625, -0.0673828125, 0.51806640625, 0.28955078125, 0.57861328125, -0.344482421875, 0.5947265625, -0.48046875, 0.6298828125, -0.1802978515625, 2.416015625, -1.578125, -0.912109375, -1.2744140625, 1.9638671875, 0.50048828125, -0.4130859375, 0.2308349609375, -0.5361328125, -0.24658203125, -1.791015625, 0.1865234375, 0.346923828125, -0.82275390625, -1.0087890625, 1.2236328125, -0.90576171875, -0.7529296875, 2.724609375, -1.9404296875, -1.63671875, -1.22265625, 0.60791015625, -1.5927734375, 0.55517578125, -0.82177734375, -1.62109375, 0.2479248046875, 0.57373046875, 0.478515625, 0.5263671875, 0.8203125, -1.02734375, 1.6455078125, 0.6943359375, -0.2032470703125, 0.44140625, -0.360107421875, -1.759765625, -0.89111328125, -0.140625, -0.497802734375, -1.173828125, 1.45703125, -2.03515625, 0.79052734375, -0.1719970703125, 1.66015625, 0.012664794921875, 0.5400390625, 0.5458984375, -0.529296875, 1.1357421875, 1.0185546875, -0.71435546875, 1.603515625, -1.333984375, 0.282958984375, 0.6083984375, -1.0576171875, -0.74462890625, 0.2486572265625, -0.556640625, 0.03448486328125, -0.67822265625, 1.0830078125, 1.72265625, -0.274169921875, -0.49560546875, -0.65673828125, 0.428466796875, -0.19189453125, -2.625, 1.7001953125, -0.72607421875, -0.19580078125, 0.66064453125, 1.0869140625, -0.452880859375, 1.98828125, 1.8837890625, 0.11212158203125, 0.1875, 0.1376953125, 1.0185546875, -0.15771484375, 0.009368896484375, -0.6240234375, 0.001163482666015625, -0.193359375, 0.171630859375, 0.9140625, -0.50048828125, -0.69677734375, 0.90771484375, 0.962890625, -0.03717041015625, 0.491455078125, 0.353759765625, -0.58203125, 0.055999755859375, -2.81640625, 0.374755859375, -0.11181640625, 0.68310546875, 1.251953125, 1.720703125, -1.5078125, 0.41455078125, 1.3037109375, 1.2080078125, 1.9296875, -0.27392578125, 0.6396484375, 0.07843017578125, -0.176025390625, 1.2353515625, -0.118408203125, 2.041015625, -0.73876953125, 1.6552734375, 1.0712890625, -1.685546875, -0.441650390625, 0.66162109375, 0.501953125, -0.7080078125, -1.759765625, -0.77490234375, 0.6904296875, -1.626953125, -0.08966064453125, 1.7607421875, 1.6943359375, 1.439453125, -1.1484375, 1.2607421875, 1.314453125, -1.4912109375, -1.130859375, 1.2626953125, -0.31396484375, -2.576171875, -0.353515625, 2.076171875, 0.935546875, -1.9267578125, 0.17041015625, -0.0787353515625, -0.403564453125, -0.2086181640625, -1.4365234375, -0.80078125, -0.00946807861328125, -0.3974609375, 0.4384765625, 0.50830078125, -0.3916015625, 0.333740234375, -1.5224609375, -0.4912109375, 1.806640625, -0.82958984375, 0.58837890625, 0.58642578125, -0.09368896484375, -0.34130859375, -2.203125, -1.29296875, 0.372314453125, -0.471435546875, -2.05078125, 0.6455078125, -1.087890625, -1.2578125, 1.634765625, 1.2392578125, 0.3017578125, 0.6357421875, 0.27001953125, -0.386474609375, 1.5869140625, 1.3232421875, -0.63330078125, 0.80126953125, 0.15771484375, -0.5361328125, 1.9306640625, 0.572265625, 0.36474609375, -0.08245849609375, -1.2265625, 0.7744140625, 1.814453125, 0.03631591796875, 1.05859375, -1.3076171875, 0.79248046875, -0.6298828125, -1.0400390625, 0.81884765625, -1.013671875, -0.384521484375, -0.63232421875, -1.5224609375, -0.2442626953125, 0.931640625, 0.8115234375, -1.5126953125, 0.170654296875, -1.2119140625, 0.1634521484375, -0.986328125, -1.7373046875, 0.7958984375, 0.61376953125, 0.7138671875, -0.43359375, -0.30615234375, 0.2296142578125, 0.1502685546875, -0.2384033203125, -0.26220703125, -0.327880859375, -0.568359375, -2.072265625, -0.8662109375, -0.86865234375, 0.1031494140625, 1.8837890625, 0.63232421875, 0.91455078125, -1.3291015625, 1.751953125, -0.497314453125, 0.1773681640625, 0.1363525390625, -1.3642578125, 0.51953125, -0.87646484375, 1.71484375, 1.8984375, -0.78955078125, 1.2529296875, -0.6806640625, 1.072265625, 0.2274169921875, 1.6123046875, -1.2646484375, -1.619140625, 1.2451171875, -0.27197265625, 1.3662109375, 0.63916015625, 0.7119140625, 0.4833984375, -0.184814453125, 1.5859375, 1.2138671875, 0.0140838623046875, 0.37060546875, 0.99462890625, -0.89697265625, 0.310302734375, 1.08984375, -1.3330078125, 1.7939453125, -0.45458984375, 1.759765625, 2.47265625, 0.407470703125, 1.0947265625, 2.71875, 0.468505859375, -1.4794921875, 1.931640625, -0.26025390625, -1.5888671875, 0.5966796875, 0.492919921875, -0.10858154296875, 0.6494140625, -0.5458984375, 0.348388671875, -0.345458984375, -0.529296875, -0.45166015625, 1.1513671875, -1.3828125, 0.061004638671875, -1.185546875, 0.91015625, 0.29931640625, 1.4130859375, 0.87744140625, 1.19921875, -1.498046875, -0.82275390625, 0.417236328125, 0.27001953125, -0.267333984375, 0.58642578125, -0.88525390625, -0.2353515625, -0.2783203125, -0.1866455078125, -1.2421875, 1.119140625, -0.263916015625, -0.53271484375, 2.037109375, -1.365234375, 0.170166015625, 0.80908203125, 0.0204620361328125, 0.71484375, 0.7392578125, 0.0103759765625, -0.1673583984375, 0.5244140625, -0.619140625, 2.087890625, -1.2451171875, 0.73974609375, 0.084716796875, 1.025390625, 0.96337890625, -1.42578125, -0.2568359375, 0.98193359375, 0.058349609375, 0.65234375, -0.34619140625, -0.2078857421875, -0.1982421875, -0.09588623046875, -0.15380859375, 0.269287109375, 0.74609375, 1.595703125, -0.52099609375, 0.1673583984375, 1.0576171875, -1.314453125, 1.3544921875, -0.1859130859375, -0.2451171875, 1.037109375, 0.253662109375, -0.5830078125, -0.513671875, -0.62255859375, -0.466796875, -1.630859375, 0.417724609375, 0.269287109375, 0.71923828125, -1.64453125, 0.654296875, 1.2041015625, -0.053436279296875, 1.236328125, -0.88330078125, -0.79052734375, -1.341796875, 1.1875, -0.8271484375, 0.043975830078125, 0.31494140625, -0.47509765625, 1.8740234375, -1.0859375, -1.1865234375, 0.86279296875, 0.8369140625, -2.359375, 0.9912109375, 0.10205078125, -1.67578125, 1.0703125, 0.497314453125, 0.98046875, -0.7861328125, -0.71142578125, 0.07659912109375, 0.5693359375, 0.479736328125, 1.73046875, -1.513671875, 0.98486328125, 1.1181640625, 1.068359375, -1.025390625, -0.57763671875, -2.423828125, -1.484375, 0.423095703125, -0.19677734375, -0.407958984375, 0.346923828125, 0.2047119140625, -1.865234375, 1.001953125, 0.0914306640625, -0.1998291015625, -0.66455078125, -2.3359375, -0.323486328125, -2.279296875, 0.413330078125, -1.9228515625, -0.030303955078125, -0.7421875, 1.234375, 0.9267578125, 0.172119140625, -0.64990234375, 0.1981201171875, 0.381591796875, 0.456298828125, 0.953125, 0.533203125, 0.0906982421875, 0.51708984375, -1.236328125, -0.95751953125, 2.501953125, 0.55224609375, 0.06768798828125, 0.0263671875, 0.25390625, 1.517578125, -1.01953125, 0.12164306640625, 1.4384765625, 0.07781982421875, 0.73974609375, 1.234375, 1.0107421875, 0.2301025390625, 0.81396484375, 0.59033203125, 1.380859375, 0.9384765625, 1.8525390625, -0.310302734375, -1.4765625, -0.1617431640625, -1.6103515625, 0.435546875, 0.6484375, 1.6416015625, 0.94921875, -1.46875, -1.0625, -1.666015625, -1.0361328125, -1.4990234375, 1.431640625, 0.10003662109375, 0.76318359375, 0.0185699462890625, 0.09075927734375, -0.387451171875, -1.033203125, 0.198974609375, 1.486328125, -1.34375, -0.33251953125, 0.06280517578125, 0.58251953125, -0.87451171875, -0.467041015625, -0.48876953125, 0.208740234375, -0.7138671875, -0.60498046875, -0.368896484375, -0.65625, -0.74853515625, 0.09033203125, 0.1851806640625, -1.1845703125, -0.03363037109375, -1.17578125, -0.7236328125, -0.1795654296875, -0.2303466796875, -0.402587890625, -2.677734375, -0.7314453125, 0.51904296875, -0.1968994140625, 0.475341796875, 0.418701171875, 1.8154296875, 0.5068359375, 0.48681640625, 0.302734375, -0.66796875, -0.86083984375, -0.495849609375, -0.72119140625, -0.4970703125, -0.796875, 1.1865234375, 0.51416015625, -0.1658935546875, 1.0478515625, 0.52880859375, 1.912109375, -0.96728515625, 0.1563720703125, 0.1282958984375, 1.3359375, 0.2216796875, 0.208251953125, 0.05352783203125, -0.51123046875, -1.439453125, 1.099609375, 0.42626953125, -2.44140625, -0.1007080078125, 0.63427734375, -0.45556640625, 0.16015625, -0.3896484375, 0.22021484375, -1.2294921875, 0.92578125, -0.164306640625, 0.044891357421875, 0.70361328125, -0.92626953125, -0.75439453125, -0.58203125, 0.09344482421875, -1.7099609375, -0.29248046875, 0.54736328125, 0.93603515625, -0.73291015625, -1.3896484375, 0.3271484375, -1.85546875, -2.931640625, 0.338134765625, -0.78076171875, -0.76025390625, -0.64404296875, -0.41748046875, -0.290283203125, 0.720703125, -0.323486328125, -0.462158203125, -0.338134765625, -1.2373046875, -1.560546875, 0.1898193359375, 0.41455078125, -1.6591796875, -0.7431640625, -0.11761474609375, -0.270263671875, -0.8056640625, -0.712890625, 2.095703125, -0.2841796875, -0.55859375, 0.141845703125, 0.5966796875, -0.8193359375, 0.7001953125, 0.91552734375, -0.50634765625, 0.0782470703125, 0.287841796875, 0.379638671875, -0.21875, 0.1116943359375, 0.10919189453125, -0.951171875, 0.47412109375, 1.1162109375, -0.537109375, -0.2900390625, 0.6416015625, -0.1351318359375, 0.65771484375, -0.4736328125, -0.222412109375, 0.78466796875, -0.6953125, -0.058746337890625, -0.470458984375, 0.3359375, -0.43896484375, -0.030120849609375, 1.4716796875, -2.134765625, 0.1439208984375, -1.7529296875, -0.241455078125, -0.8193359375, 0.254638671875, 1.83984375, -1.830078125, 0.273193359375, -0.5283203125, -0.26171875, 2.478515625, 2.080078125, 1.3037109375, 0.1387939453125, -0.55078125, 0.2210693359375, 0.106689453125, -1.2783203125, 0.818359375, -0.99853515625, -1.3681640625, -0.3017578125, 0.595703125, 0.91845703125, 1.5146484375, 0.8544921875, -0.283203125, 0.1956787109375, 0.411376953125, -0.447265625, -0.11627197265625, 1.6923828125, -0.80908203125, -0.11993408203125, 0.95751953125, 1.4541015625, 0.87744140625, -0.028839111328125, 1.0390625, -0.23193359375, 0.7666015625, -0.314453125, -1.1552734375, -1.83203125, -0.45849609375, -0.7314453125, 0.46337890625, -1.5908203125, -0.8291015625, -0.72705078125, -1.06640625, -1.5087890625, 0.70849609375, 0.310302734375, 0.266357421875, 0.61376953125, 0.00661468505859375, -2.390625, 1.0625, 1.2578125, -0.7138671875, -0.1395263671875, -0.744140625, -0.1915283203125, -0.12451171875, -2.04296875, -0.0968017578125, 1.162109375, 0.281494140625, -0.346435546875, -0.95458984375, 0.0310821533203125, 0.0859375, 0.9306640625, -0.11761474609375, -0.1629638671875, -1.6318359375, 0.140869140625, -0.6953125, -0.5, 0.57177734375, -0.49853515625, 1.0869140625, 0.484619140625, -0.0166473388671875, 0.173583984375, -1.5439453125, 0.53369140625, 0.84375, 1.6103515625, 1.005859375, -0.9482421875, -1.525390625, 0.85986328125, 0.1942138671875, 0.67822265625, -0.27880859375, -0.90771484375, 0.5, 0.87451171875, -0.048187255859375, 0.1383056640625, 0.403076171875, 1.2333984375, -2.02734375, 0.19775390625, 0.5390625, 0.92626953125, -0.76953125, 1.3427734375, -1.150390625, 0.10943603515625, -1.2099609375, -0.7880859375, 0.59130859375, 0.12890625, -0.7900390625, -1.310546875, -0.326904296875, 0.361083984375, 0.41650390625, -0.221435546875, 2.03515625, 0.0570068359375, 0.875, -1.1396484375, 0.54736328125, -2.00390625, -2.83203125, 0.95947265625, -0.005126953125, -0.1361083984375, 0.10400390625, -0.5087890625, 1.013671875, 0.60498046875, 0.1761474609375, 0.9609375, -0.50244140625, 0.79443359375, 1.16796875, 1.474609375, -1.2021484375, -0.046356201171875, 0.9052734375, -0.44384765625, 0.2900390625, 1.1884765625, 0.435546875, -2.572265625, 0.908203125, 1.7236328125, 0.1461181640625, 0.23876953125, 1.2099609375, -2.658203125, -0.82373046875, 0.71240234375, -1.1142578125, 0.07061767578125, -2.18359375, -0.35205078125, 0.470703125, -0.321044921875, 0.1278076171875, -0.94775390625, -1.1767578125, -0.1051025390625, -0.943359375, 0.7158203125, -1.494140625, 1.697265625, -1.0478515625, -0.18017578125, 0.3515625, -0.027252197265625, 0.88427734375, -0.11175537109375, 0.396484375, 0.396728515625, 0.79345703125, -0.8681640625, 0.161865234375, 0.66162109375, 0.07452392578125, 3.123046875, 0.214599609375, -2.16015625, 0.464599609375, -0.1578369140625, -1.203125, 0.72216796875, -1.1728515625, 0.3203125, -0.53662109375, -0.1932373046875, -0.63818359375, 0.5869140625, 1.08203125, -0.6474609375, 0.1932373046875, -0.050811767578125, 2.90234375, 0.1468505859375, -0.2900390625, -2.73828125, -0.9267578125, 1.2451171875, 0.4599609375, 0.80029296875, 0.171142578125, 0.3564453125, -0.74755859375, -0.36083984375, 1.36328125, 0.65185546875, -1.3974609375, 0.9345703125, -0.81494140625, 1.623046875, 0.4423828125, 0.04888916015625, 0.55908203125, -0.91650390625, 0.8681640625, -0.5166015625, -1.4755859375, 1.2265625, -1.0791015625, -1.671875, 1.181640625, -0.1441650390625, 1.0810546875, -0.10931396484375, 0.53271484375, 1.478515625, -0.18701171875, -1.328125, -1.8037109375, 0.80078125, 0.60400390625, -0.97509765625, -1.2998046875, -0.74609375, 0.99365234375, -0.0260009765625, -0.45361328125, 1.705078125, 0.0843505859375, 0.6923828125, 0.91845703125, -1.740234375, 0.73681640625, -0.56787109375, -1.771484375, 1.8486328125, 0.70947265625, 0.93505859375, -0.09228515625, -1.337890625, 0.0687255859375, 1.087890625, 1.494140625, -0.475830078125, -0.2783203125, -0.0684814453125, 0.1607666015625, 0.260986328125, 0.80908203125, 0.61474609375, 0.5673828125, -1.876953125, 1.4296875, 0.7451171875, 0.82666015625, 0.86669921875, -0.96435546875, -1.1806640625, 1.2978515625, -0.0831298828125, 1.3720703125, 0.70361328125, 1.046875, -1.587890625, -2.986328125, -0.0816650390625, 0.012542724609375, -0.771484375, 0.97412109375, -0.611328125, 1.4423828125, 0.65478515625, 0.072021484375, 0.2369384765625, -1.9619140625, -0.7802734375, -0.58203125, 0.6982421875, -1.5546875, 0.412353515625, 0.544921875, 0.87109375, 1.0927734375, -0.5322265625, 1.3662109375, 4.765625, -1.3515625, -0.323486328125, -0.35888671875, -0.4775390625, 0.18798828125, -0.62890625, -0.320556640625, -0.99267578125, 0.64794921875, -0.035919189453125, 1.193359375, 0.29345703125, -0.357421875, 1.080078125, 0.0830078125, 1.458984375, -1.751953125, 0.9716796875, -1.017578125, 0.68798828125, -0.08203125, 0.45556640625, 0.66455078125, 0.6962890625, 0.1395263671875, -2.04296875, -0.72900390625, 2.787109375, -0.396484375, 0.280517578125, -0.0885009765625, -1.8544921875, 0.568359375, -1.7080078125, -0.88427734375, -0.21923828125, 0.5234375, -1.9892578125, -1.814453125, 0.72802734375, -0.4951171875, -1.0107421875, -0.4951171875, -0.403564453125, -0.038177490234375, -0.66357421875, 0.72607421875, -0.3427734375, -2.16796875, 0.2113037109375, 1.138671875, -0.8564453125, 0.498046875, -0.89111328125, -0.208251953125, 0.181640625, -0.3369140625, 0.73876953125, 0.258056640625, 0.153564453125, -0.6513671875, 0.492919921875, -0.7080078125, -0.472412109375, 0.1907958984375, -1.7880859375, -0.2421875, -1.861328125, -1.783203125, 0.10760498046875, 1.40625, -0.441162109375, -1.849609375, -1.0322265625, 1.6640625, 0.62451171875, -0.89501953125, -0.385498046875, -0.48876953125, -0.52490234375, 1.5361328125, -1.7333984375, -0.29248046875, -0.77880859375, 0.61279296875, 0.19677734375, 0.11688232421875, -0.548828125, 0.52490234375, -0.1500244140625, -1.80859375, -0.06292724609375, 0.75537109375, 0.0601806640625, -0.6044921875, 0.435302734375, -0.457763671875, -1.4462890625, -0.74755859375, -0.288818359375, 1.1533203125, 1.5986328125, 0.280517578125, -1.0595703125, 0.37109375, -1.869140625, 0.497802734375, -0.1717529296875, 0.43505859375, 0.09478759765625, -0.544921875, 0.178466796875, 2.142578125, 0.6748046875, -0.9482421875, 0.93505859375, 0.7919921875, -0.14404296875, -0.876953125, -0.9619140625, -0.099365234375, 0.61376953125, -0.84326171875, -0.08782958984375, 0.62060546875, 1.009765625, -0.5205078125, 0.40234375, -1.740234375, 0.258544921875, -0.40234375, -0.1383056640625, -1.86328125, -0.044403076171875, -1.126953125, -1.080078125, 0.277587890625, 0.62158203125, 0.65673828125, 0.3720703125, -0.432861328125, -0.58349609375, -0.62060546875, -0.22021484375, -0.05462646484375, -0.53173828125, -0.461669921875, 0.84326171875, 0.033538818359375, 0.7529296875, -0.32763671875, -1.060546875, -0.14501953125, 0.30517578125, 1.708984375, -0.5654296875, 2.052734375, -0.434326171875, 0.08837890625, 2.3046875, 0.369384765625, 1.0634765625, -1.841796875, -0.373046875, 0.102783203125, -0.98291015625, 0.505859375, -0.68212890625, 0.2203369140625, 0.5146484375, 0.0188751220703125, 0.2802734375, 0.06341552734375, -1.0556640625, -1.0810546875, 0.19384765625, 1.0029296875, 0.24609375, -0.1412353515625, -0.5869140625, -1.5595703125, 0.038909912109375, 0.58740234375, -0.318603515625, 0.5595703125, 0.2059326171875, 0.496826171875, 0.432373046875, -0.402587890625, 2.544921875, 0.18505859375, -0.0693359375, 0.366455078125, 2.060546875, 0.43994140625, 0.1956787109375, 0.115234375, -1.3994140625, 0.9677734375, 0.8310546875, -1.6044921875, 0.91552734375, -0.50830078125, -0.85546875, -2.125, -0.2451171875, -0.1910400390625, -0.57861328125, 1.8837890625, 0.344482421875, 0.96044921875, -0.36083984375, 0.82666015625, -0.344482421875, 1.04296875, 0.8017578125, 1.5517578125, 0.2227783203125, -0.121826171875, 1.8115234375, -0.64404296875, -1.005859375, -0.11016845703125, 0.11279296875, 0.78271484375, -0.428955078125, 0.57080078125, 1.1884765625, -1.3193359375, 0.3466796875, 1.556640625, -1.41015625, -0.432373046875, -0.3818359375, 0.998046875, 1.5126953125, -1.42578125, -0.8359375, -0.459716796875, -0.5, 2.193359375, 0.54736328125, -1.47265625, -0.92529296875, 1.1865234375, 0.47607421875, -0.051544189453125, -1.16015625, -0.92431640625, 0.64013671875, -0.8701171875, 0.357421875, 0.1060791015625, 0.6435546875, 0.30810546875, 0.7900390625, 0.81591796875, 0.451904296875, -0.6416015625, -1.375, -1.1904296875, 0.53564453125, -1.7373046875, 0.8095703125, 3.068359375, -0.91748046875, -0.552734375, 0.59619140625, 0.29638671875, -2.193359375, -0.2225341796875, -0.2205810546875, 0.042266845703125, 0.58349609375, -1.380859375, 0.416015625, -1.2177734375, -1.2919921875, 0.90966796875, -0.7216796875, 1.8193359375, 0.0897216796875, 0.0295867919921875, 0.06317138671875, 0.365966796875, 1.20703125, -0.55517578125, 1.84765625, 0.7705078125, 0.61474609375, -0.37060546875, -0.53759765625, -1.513671875, 0.325439453125, 0.166015625, 0.15087890625, -0.38134765625, 0.3349609375, 1.6162109375, -1.021484375, 0.033233642578125, 2.076171875, 0.82080078125, -2.595703125, -0.444580078125, 1.005859375, -0.444580078125, 0.859375, -0.39990234375, -1.0322265625, -2.021484375, 0.00989532470703125, 1.455078125, 0.642578125, -1.5400390625, 0.490966796875, 0.1414794921875, 0.334228515625, -0.82421875, -1.81640625, -1.5986328125, 1.5517578125, -0.47412109375, 1.2646484375, -2.18359375, -0.2298583984375, 1.4580078125, 1.32421875, 1.48046875, -0.2000732421875, -1.7763671875, 1.1669921875, 0.47314453125, 0.1529541015625, 1.400390625, 1.1435546875, -0.208740234375, -1.443359375, -0.181640625, 0.31787109375, -0.76220703125, -2.302734375, -0.54443359375, -0.260498046875, 0.447998046875, -0.48974609375, 0.6708984375, 0.8798828125, -0.059783935546875, 1.0751953125, -1.0146484375, 0.78564453125, 0.086181640625, 0.432373046875, -0.68115234375, 2.23828125, 1.08203125, 0.71728515625, -0.9521484375, 0.491943359375, 2.791015625, -0.69091796875, -0.94775390625, 1.189453125, 0.1796875, -0.047943115234375, 1.458984375, -0.75927734375, 1.8515625, -1.08984375, -0.77734375, 0.429931640625, -0.9306640625, -0.533203125, 0.63720703125, 0.37939453125, 0.5458984375, 0.75439453125, -0.4892578125, 0.336669921875, -0.3974609375, 0.69873046875, 1.740234375, 0.69677734375, 0.53076171875, 1.609375, 0.703125, -1.267578125, -0.564453125, 0.1920166015625, 1.78515625, -0.327392578125, -0.36376953125, 0.15966796875, -0.466796875, 0.5439453125, 0.093505859375, 0.04705810546875, -0.10882568359375, 1.1015625, -0.916015625, -0.49365234375, 0.292236328125, 0.943359375, -0.54931640625, -2.03125, -1.3173828125, 0.441650390625, -1.9892578125, 2.955078125, 0.8896484375, -0.68359375, 1.0849609375, -0.3837890625, -1.7509765625, 1.4169921875, -0.1671142578125, -2.0703125, -1.779296875, 0.6923828125, -0.93896484375, 1.8369140625, 0.402587890625, -0.2333984375, 3.8683414459228516e-05, -0.5703125, 1.5986328125, -0.76806640625, -1.5166015625, 0.67236328125, -0.6728515625, -1.595703125, -1.1787109375, 0.875, -0.28173828125, -0.17626953125, 1.3251953125, -0.342041015625, -0.81298828125, -0.04595947265625, 1.30078125, 2.220703125, -0.2476806640625, -0.3544921875, 0.381591796875, -0.1632080078125, 1.7529296875, 0.587890625, -0.443115234375, -0.96142578125, 0.7734375, -1.162109375, -0.7470703125, -0.0154266357421875, 0.235595703125, 0.49169921875, 0.4931640625, -0.0682373046875, -1.0078125, -0.5625, 0.2841796875, 0.10870361328125, 0.60009765625, -0.323486328125, 0.74072265625, -0.1102294921875, 1.203125, 2.01953125, 0.474853515625, -1.240234375, -0.445068359375, -1.513671875, -0.09112548828125, 0.62841796875, -2.7734375, 0.64453125, -0.73828125, -1.7001953125, -0.904296875, -1.8310546875, -1.3681640625, 0.63671875, 0.10858154296875, -0.21875, 1.556640625, -0.150146484375, -0.375732421875, -0.1572265625, -1.4267578125, -0.779296875, 0.076416015625, 0.6884765625, 0.2174072265625, -0.462646484375, 0.412353515625, 0.58251953125, 0.7001953125, 0.72265625, 0.90673828125, -0.87158203125, -1.1025390625, 0.55419921875, -0.66845703125, -0.6337890625, 0.52001953125, -0.047393798828125, 0.432373046875, 1.7626953125, 1.0771484375, 0.27880859375, 0.279296875, -0.6435546875, -0.11639404296875, -0.01061248779296875, 0.09515380859375, 0.471435546875, -0.5341796875, 0.219482421875, 0.68701171875, -0.72900390625, -1.4755859375, 1.3681640625, 0.54296875, 0.261962890625, 2.01953125, 1.2451171875, -1.4833984375, -0.638671875, 0.1104736328125, 0.66845703125, 0.41162109375, -1.234375, 0.60791015625, -0.266845703125, -2.041015625, -0.89599609375, 1.5830078125, -1.185546875, 0.412841796875, -1.42578125, -1.6376953125, 0.255859375, 0.64208984375, 0.371826171875, 1.1865234375, -1.3642578125, -1.9443359375, 0.286376953125, -0.6728515625, -0.472900390625, -0.263427734375, 1.2919921875, -0.771484375, -1.546875, -0.283447265625, -0.72998046875, 2.65234375, -0.31982421875, -1.3759765625, 0.085693359375, 1.55078125, 0.34814453125, 0.6328125, -0.708984375, 1.8291015625, 0.6337890625, 0.70556640625, 0.6748046875, 2.728515625, -1.6240234375, 0.7939453125, -0.09503173828125, 0.489501953125, -0.7099609375, -0.09063720703125, 0.453125, 0.1614990234375, -0.62109375, 0.459716796875, 0.498046875, 0.279541015625, -0.470947265625, -0.60791015625, 0.2049560546875, -1.0615234375, -1.337890625, 0.71533203125, 1.8291015625, -0.438720703125, -2.126953125, -0.054168701171875, 0.9794921875, 0.5146484375, -0.3603515625, -0.135498046875, 0.2344970703125, 0.94189453125, 0.88720703125, 0.8525390625, -0.53271484375, 0.322998046875, -1.375, -2.033203125, -0.374755859375, -0.556640625, -0.1988525390625, 0.29052734375, -0.232666015625, -0.281005859375, 0.368408203125, 0.20751953125, -0.1956787109375, 0.548828125, 1.265625, 0.350830078125, -0.29833984375, 0.365234375, -0.048248291015625, 0.9990234375, 1.083984375, 1.3564453125, 0.77099609375, 1.0322265625, -0.197265625, -0.74169921875, -0.9384765625, -0.369873046875, 2.267578125, -1.369140625, -0.18017578125, 1.1123046875, -0.97412109375, -0.4609375, -1.3828125, 0.55859375, 0.242919921875, 1.412109375, 0.55810546875, 0.2491455078125, 1.611328125, -1.5146484375, -0.43115234375, 0.6982421875, 0.46533203125, -0.2919921875, -1.451171875, -0.289794921875, -1.4169921875, -0.470703125, 0.239501953125, -0.399658203125, -0.70166015625, 0.376708984375, 1.595703125, -1.9140625, 0.80126953125, 1.3701171875, 2.216796875, 0.318603515625, -0.387451171875, 1.564453125, -0.8408203125, -1.5166015625, 1.4267578125, -0.1920166015625, -0.775390625, 0.27099609375, 0.38330078125, -0.4638671875, -0.368408203125, -0.90966796875, 0.92626953125, 1.4912109375, -0.70751953125, -0.289794921875, 0.57568359375, -1.1416015625, -0.1595458984375, 0.0088653564453125, 1.46484375, 0.465087890625, -2.935546875, -3.015625, 0.50927734375, -0.0631103515625, 0.026458740234375, -0.456787109375, 0.64892578125, -1.423828125, 0.9091796875, 0.045318603515625, -0.646484375, 0.5703125, -0.321533203125, -2.0859375, 1.546875, 1.845703125, 1.7783203125, 0.0960693359375, -1.7880859375, -1.0849609375, 0.60009765625, 0.1124267578125, 0.07598876953125, 1.8095703125, -0.02978515625, 1.814453125, 1.4580078125, 1.19921875, -0.116943359375, 0.170166015625, -1.603515625, 0.607421875, -1.1376953125, 1.4326171875, -1.5146484375, 0.0248260498046875, -0.939453125, 0.55126953125, -2.361328125, 0.05535888671875, 0.5048828125, 0.301025390625, 0.428955078125, 0.26220703125, 0.02685546875, -0.62744140625, 0.5693359375, 0.10107421875, 0.289306640625, 0.9287109375, -0.32470703125, 0.476806640625, -1.8955078125, 0.6025390625, -1.2978515625, -0.11688232421875, 1.375, -1.5556640625, 0.26611328125, 0.76318359375, 0.572265625, 0.52001953125, -2.150390625, 0.75830078125, -0.97265625, -0.38916015625, 1.310546875, -0.369873046875, -0.53662109375, -1.150390625, -0.2491455078125, 0.97314453125, 0.50439453125, 1.8603515625, -0.06427001953125, -0.72607421875, -0.051849365234375, -0.8232421875, 0.260498046875, -0.306396484375, -0.48583984375, 1.642578125, -0.470703125, -0.315673828125, -0.427734375, -0.93603515625, -0.0841064453125, -0.2244873046875, 0.9453125, -0.7578125, -1.2646484375, 0.6259765625, 4.06640625, 0.375244140625, -2.568359375, 0.237060546875, -0.83154296875, 0.6298828125, 1.919921875, -0.31884765625, -0.347412109375, -0.7255859375, -0.66552734375, -1.357421875, -0.83447265625, -0.0386962890625, 0.391845703125, -1.62890625, 0.65087890625, 0.54248046875, 0.0262451171875, -1.349609375, -0.06646728515625, -0.56298828125, -0.125732421875, -2.0546875, -0.6865234375, 1.171875, 0.736328125, -0.70263671875, -0.6552734375, -0.7255859375, -1.7158203125, 0.272705078125, -1.9599609375, -0.37109375, -0.499755859375, 1.7744140625, -2.03515625, -1.4677734375, -1.455078125, -0.859375, 1.0400390625, -0.37255859375, 0.59130859375, 0.5576171875, -0.57177734375, 0.90283203125, -0.260986328125, -0.67041015625, 0.599609375, 1.4658203125, 1.02734375, -0.92529296875, -2.6640625, -0.07720947265625, -0.93994140625, 1.380859375, 1.8017578125, 0.423095703125, 0.4375, 0.68701171875, 0.07879638671875, -0.09271240234375, -0.66162109375, 2.025390625, -1.26171875, -0.74365234375, 0.454345703125, -0.36767578125, 0.05523681640625, 0.5126953125, 0.6533203125, 0.537109375, -1.92578125, -0.351318359375, 0.76953125, 0.5830078125, -2.556640625, 1.2412109375, 0.3134765625, -0.95068359375, -1.017578125, -1.7001953125, 1.9912109375, -2.466796875, 0.60595703125, -0.060302734375, -1.6689453125, -0.42333984375, -1.021484375, -1.0234375, 0.962890625, 0.8984375, 0.01107025146484375, 0.69384765625, -1.1611328125, -0.546875, -1.1748046875, -2.248046875, 0.8525390625, -1.3115234375, -0.47998046875, -0.42578125, -0.441650390625, 0.88232421875, -0.1729736328125, 1.67578125, 0.043243408203125, 1.5478515625, -0.29638671875, -0.5341796875, -0.9033203125, -0.1544189453125, 1.482421875, -0.328125, 2.23046875, 1.0185546875, 0.125, 1.0068359375, 2.185546875, 1.2314453125, -0.82666015625, -0.72021484375, 0.6435546875, -1.693359375, -1.87109375, 1.7978515625, 0.273681640625, -1.21484375, 0.732421875, -0.52490234375, -1.0546875, -0.505859375, -0.17626953125, 0.66162109375, 0.7568359375, -1.6123046875, -2.91015625, -0.091796875, -1.662109375, 0.4833984375, -1.46484375, -1.1953125, -2.32421875, 0.05950927734375, 0.223876953125, -1.072265625, -0.287109375, 0.52392578125, 0.99560546875, 0.409912109375, 0.210693359375, -0.6240234375, 0.86865234375, -1.171875, -0.1942138671875, -1.6435546875, -0.41748046875, 0.35498046875, -0.966796875, -1.8916015625, -0.456787109375, -1.521484375, 0.970703125, 0.79345703125, 0.4833984375, 0.379150390625, 0.5751953125, -0.415771484375, -0.66796875, 2.3671875, 0.78515625, 0.1719970703125, -1.076171875, 2.18359375, -0.0604248046875, 1.681640625, -1.41796875, 1.0927734375, -0.207275390625, 0.669921875, 2.333984375, -0.32861328125, -0.6123046875, -2.294921875, 1.4052734375, 1.4375, 2.84765625, 0.431640625, -0.414794921875, 0.260498046875, 0.30322265625, -1.134765625, -0.6240234375, -1.23828125, 1.9169921875, 2.693359375, 0.84521484375, -0.52392578125, -1.8701171875, -2.283203125, -0.826171875, 0.9482421875, 0.022247314453125, 0.78857421875, 0.086669921875, 1.130859375, -1.44140625, 1.7236328125, -1.49609375, -0.599609375, -1.541015625, 0.1844482421875, -0.244873046875, 0.487548828125, -0.5087890625, 0.73095703125, -2.189453125, -0.55859375, 1.099609375, -1.134765625, -1.6455078125, 0.5537109375, 1.0341796875, 0.435302734375, -1.1533203125, 1.3056640625, -0.51318359375, -0.65771484375, -0.0833740234375, 2.203125, -1.0400390625, 0.759765625, 1.2900390625, 0.6923828125, 1.134765625, -0.84716796875, 0.14794921875, -1.7900390625, -1.2421875, -2.095703125, 0.74609375, 0.30029296875, 0.491455078125, -1.265625, 0.57373046875, 0.10333251953125, 0.05938720703125, 0.732421875, -0.3505859375, -1.5654296875, -0.2110595703125, -0.06591796875, 0.52099609375, 0.409912109375, 1.59375, -1.013671875, -0.4248046875, 1.8525390625, 1.484375, -0.42138671875, 1.2509765625, 0.96337890625, 1.6162109375, 1.1357421875, 0.138916015625, 0.88134765625, -0.1600341796875, -0.324951171875, -0.129638671875, 1.2783203125, 0.91943359375, -0.186279296875, -0.229736328125, -1.158203125, 0.2288818359375, 0.480712890625, 1.837890625, -0.58544921875, -0.296630859375, 0.44287109375, 1.6572265625, 2.052734375, -1.7734375, 0.8271484375, -0.76513671875, 1.5517578125, -1.568359375, -0.72509765625, 0.40478515625, 0.73681640625, -1.0517578125, -0.030731201171875, -0.4619140625, 0.339111328125, -0.53369140625, 0.71142578125, -0.5546875, -0.42529296875, -0.0022907257080078125, -0.29150390625, -0.53564453125, 1.5283203125, 0.3017578125, -0.68896484375, -0.148193359375, 1.2646484375, -0.044586181640625, -1.982421875, -1.203125, -0.79833984375, 0.7509765625, 0.07373046875, -0.0027942657470703125, -0.50634765625, -0.55810546875, 1.21875, 0.350830078125, -0.3671875, -1.697265625, 0.03350830078125, -0.85693359375, -0.92333984375, -0.642578125, -0.84326171875, -0.260986328125, -0.50048828125, 0.45703125, -1.5732421875, -0.9912109375, 1.607421875, 1.33203125, 1.96484375, -0.13330078125, 0.6796875, -0.1763916015625, -0.2744140625, -0.56982421875, -1.0068359375, -1.724609375, -0.1373291015625, -0.2330322265625, 1.0634765625, -1.7001953125, 0.04345703125, -0.144287109375, -1.2197265625, -0.16259765625, -0.5703125, -1.0888671875, -2.041015625, -1.1328125, -0.86376953125, -0.27099609375, -1.80859375, -1.142578125, -1.2548828125, -1.5205078125, 0.54736328125, -0.373779296875, -0.8544921875, 1.953125, -0.79150390625, 0.68115234375, -0.287841796875, 0.982421875, 0.43310546875, -0.97998046875, 1.1630859375, 1.2373046875, 0.8720703125, 0.0227813720703125, -0.0157318115234375, -0.409423828125, -0.5126953125, 0.1737060546875, 0.10308837890625, -0.360107421875, -0.6552734375, 1.740234375, -0.57861328125, -0.859375, 0.59765625, 0.10491943359375, -0.412109375, -1.1435546875, -1.189453125, 0.1683349609375, 0.1151123046875, -0.404052734375, -0.1812744140625, -1.408203125, 2.025390625, -0.08697509765625, 0.4296875, -0.6630859375, 0.30224609375, 1.23828125, 0.55859375, -0.5634765625, -0.251953125, -0.5859375, -0.434326171875, -0.155517578125, -0.73974609375, 1.5908203125, 0.5927734375, 0.264892578125, -1.51171875, -1.4814453125, -0.11431884765625, -0.7529296875, -0.5791015625, -1.4013671875, 0.900390625, 1.0400390625, 0.37744140625, -1.654296875, 0.259521484375, -0.71142578125, 0.1829833984375, -0.07000732421875, -1.4208984375, 2.3203125, -0.034576416015625, -0.521484375, -1.408203125, 2.0078125, 0.69921875, 1.220703125, -0.77685546875, -0.69189453125, 0.0513916015625, -0.50537109375, 1.2158203125, 0.96875, 0.072021484375, 0.1944580078125, 0.9599609375, -0.041717529296875, 1.958984375, -2.048828125, 0.35546875, 1.0478515625, -0.8642578125, 0.06744384765625, 1.919921875, -0.498779296875, -0.0265045166015625, -0.3740234375, 0.1363525390625, 0.619140625, 0.271728515625, 0.0268707275390625, -0.62939453125, -0.70263671875, 0.458984375, 1.7001953125, 1.67578125, -0.291748046875, 1.025390625, 1.1337890625, 1.2099609375, -0.59619140625, 1.3017578125, 1.3427734375, -0.86328125, -0.55322265625, -0.568359375, 0.79541015625, 0.93310546875, -2.03125, -0.97021484375, 0.99267578125, -1.4189453125, -0.8505859375, 0.888671875, 0.44873046875, -0.9052734375, 0.220947265625, -0.34423828125, 0.30908203125, -1.6416015625, -0.56396484375, 0.369873046875, -0.14453125, -0.9560546875, -2.33203125, -1.2275390625, -0.7666015625, 0.67431640625, 0.032745361328125, 2.009765625, 1.080078125, 0.44580078125, 0.0207366943359375, 0.2548828125, -0.52392578125, 1.923828125, -0.2294921875, 1.1240234375, -1.11328125, 1.3818359375, -0.87353515625, 0.130615234375, 1.1162109375, -0.73095703125, 1.6494140625, -1.0576171875, -1.521484375, -0.434814453125, 0.82958984375, -1.423828125, 0.89306640625, 0.845703125, 1.953125, -0.21923828125, -0.250732421875, 0.44140625, 0.80419921875, 0.84033203125, 0.669921875, 1.5576171875, 0.45751953125, -0.982421875, 0.081298828125, 0.80859375, -1.255859375, 0.195068359375, -0.39599609375, 1.5029296875, -0.356201171875, 1.4677734375, -1.3994140625, 0.787109375, -0.8759765625, 1.109375, 0.48876953125, -1.18359375, 1.30078125, 0.65087890625, 0.62109375, -0.06890869140625, 0.87255859375, 0.81689453125, 0.0791015625, -0.4765625, 1.0517578125, -1.2666015625, 1.2734375, -1.0791015625, 0.65478515625, 1.0546875, 0.55078125, 1.5693359375, 0.06402587890625, -0.6376953125, 1.759765625, -0.4111328125, -0.70556640625, -0.34814453125, -1.8203125, -0.38818359375, 2.4453125, 0.71630859375, -0.11236572265625, 0.0789794921875, -0.19775390625, 1.4267578125, -2.560546875, -0.298583984375, -1.1611328125, -0.481201171875, 0.2132568359375, -1.0869140625, 0.002300262451171875, 0.2139892578125, -0.174560546875, 1.267578125, 0.267333984375, -0.86669921875, -0.7177734375, -0.252197265625, 0.1517333984375, 1.4599609375, 0.4599609375, 0.38330078125, 1.435546875, 0.11431884765625, 0.82666015625, 0.1387939453125, 0.18408203125, 0.95263671875, 1.685546875, 0.0570068359375, 0.26025390625, 1.4345703125, 1.19921875, -0.83642578125, -0.373779296875, -0.250244140625, 0.8623046875, -0.83056640625, -0.3232421875, 1.369140625, -0.1612548828125, -0.853515625, -0.055328369140625, 0.412353515625, -1.3369140625, -0.4345703125, -0.599609375, 1.1962890625, -0.381591796875, -1.099609375, 1.4951171875, 1.2685546875, 0.050872802734375, -1.7001953125, -0.87890625, 0.5322265625, -1.2783203125, -1.244140625, -0.859375, -1.7802734375, -2.2578125, 0.8564453125, 0.015716552734375, 0.07183837890625, 0.400634765625, 0.408203125, 0.89013671875, -0.98876953125, -1.4658203125, -0.241943359375, -0.68115234375, 0.242431640625, 1.244140625, -1.822265625, 0.471435546875, -1.4072265625, -0.12060546875, -0.199951171875, -0.71484375, 0.306640625, 0.58544921875, -1.2939453125, 1.693359375, -1.6044921875, 1.0595703125, 2.599609375, -0.5703125, 0.5146484375, -0.07763671875, -0.2271728515625, 0.1976318359375, 1.63671875, 0.1949462890625, 1.1220703125, -0.021087646484375, -0.83447265625, -1.068359375, -1.703125, -0.312744140625, 0.740234375, 0.087158203125, -0.53759765625, 0.962890625, 0.92333984375, -1.1005859375, 0.003948211669921875, 0.2010498046875, -0.84423828125, 0.04510498046875, -0.5283203125, 0.2364501953125, -0.5166015625, -1.5205078125, 0.67578125, -0.96142578125, -0.309814453125, 0.6259765625, -1.1435546875, -0.90576171875, -1.318359375, 0.84912109375, 0.912109375, 0.12890625, 0.440185546875, -0.91064453125, -1.9013671875, -0.78564453125, 0.60546875, 0.311279296875, 0.054901123046875, 0.6259765625, -0.92529296875, -0.85546875, -0.35498046875, -1.8720703125, -1.625, 0.10797119140625, 0.72607421875, -0.81787109375, -1.279296875, 0.0196075439453125, -0.12939453125, 0.5947265625, 0.10821533203125, -0.927734375, -1.2822265625, 1.3857421875, -0.6259765625, -0.9970703125, 0.9873046875, 0.349365234375, 0.896484375, 0.321533203125, -0.29150390625, -0.220703125, 0.463623046875, 0.08905029296875, -1.3857421875, 0.327392578125, -0.4580078125, 0.64892578125, -1.22265625, -0.8642578125, 0.02984619140625, -0.044525146484375, -0.3310546875, -2.474609375, 0.75830078125, -0.98193359375, -0.650390625, -0.015655517578125, 0.71337890625, 0.195556640625, 0.450927734375, 0.6572265625, -0.41650390625, -0.9755859375, 0.748046875, 1.1904296875, -0.11895751953125, -0.50634765625, 0.1953125, -0.9375, -0.259033203125, -0.2252197265625, -0.78662109375, -0.53076171875, 2.619140625, 0.74658203125, -1.8525390625, -0.33935546875, 0.333984375, 0.10784912109375, 1.9384765625, 1.796875, 0.412353515625, 0.1715087890625, -1.083984375, 0.4619140625, 1.7353515625, -0.677734375, -0.08074951171875, -0.830078125, 0.7509765625, -2.796875, 0.2054443359375, 0.880859375, 0.474365234375, 0.1649169921875, 0.48779296875, 1.123046875, 0.1602783203125, -0.59033203125, -1.224609375, 0.339111328125, 1.2783203125, -0.7578125, 0.92333984375, -0.69091796875, -2.205078125, -1.583984375, -0.5546875, -1.3720703125, 0.548828125, 1.0087890625, 0.7451171875, 0.45166015625, -0.2044677734375, 2.21875, -1.2294921875, -0.35546875, -0.25341796875, 1.5205078125, -0.439453125, -2.802734375, 2.396484375, -0.81689453125, -0.064208984375, 0.83642578125, -0.387451171875, 0.64794921875, 0.6064453125, 0.85498046875, 0.1788330078125, -0.2283935546875, -0.99169921875, 0.046142578125, 1.576171875, 0.56005859375, 0.96484375, 0.53515625, -1.1640625, -1.220703125, -0.87060546875, 1.802734375, 0.0279541015625, 0.57470703125, -0.10552978515625, 1.0625, -0.23095703125, 0.802734375, -1.126953125, 0.18994140625, -1.2744140625, 0.81982421875, 0.494140625, 0.046783447265625, -0.47900390625, -0.339599609375, 0.65673828125, 0.513671875, -3.396484375, -0.449951171875, -0.30322265625, -2.337890625, -1.0908203125, 1.060546875, -0.58056640625, -0.499267578125, 1.5205078125, 0.9765625, -0.52734375, -0.66357421875, -0.78369140625, -0.0693359375, 0.458740234375, -0.221923828125, -0.0273284912109375, -0.2705078125, -2.2890625, -1.810546875, -0.38232421875, -0.261962890625, -0.35888671875, -0.2607421875, -1.9873046875, 1.0107421875, 0.0887451171875, -0.69677734375, -1.572265625, 1.419921875, -1.283203125, 1.2275390625, -0.681640625, -1.326171875, -0.07012939453125, -0.051605224609375, 0.270263671875, -0.63818359375, 0.11968994140625, 0.22412109375, -0.83447265625, 1.65234375, 0.83154296875, 0.2103271484375, 0.73974609375, 0.58056640625, -0.796875, -0.190673828125, -1.22265625, 0.277587890625, 0.650390625, -0.6787109375, 0.0672607421875, 1.0966796875, -0.71044921875, -0.798828125, 1.2529296875, -1.162109375, -0.10064697265625, 0.1724853515625, 0.73193359375, -0.3154296875, -0.42138671875, 0.56982421875, -0.71728515625, -0.308349609375, -0.010406494140625, -1.7275390625, 0.7353515625, -0.79443359375, 0.3251953125, 0.67578125, -0.6552734375, 0.8203125, 0.951171875, -1.197265625, 1.9765625, -1.2607421875, 0.6044921875, -0.4072265625, -0.58056640625, -0.401123046875, -0.65283203125, 1.1650390625, 1.2421875, -0.52685546875, -0.56494140625, -0.98583984375, 0.0467529296875, 0.5673828125, 0.04412841796875, -0.732421875, -1.482421875, 1.4853515625, -0.35791015625, 1.953125, -0.321533203125, 0.70849609375, 0.76953125, 0.72021484375, -0.461669921875, 1.4296875, -0.3388671875, -0.76806640625, 0.80859375, -0.6328125, -0.58740234375, 1.1875, 0.83837890625, -0.03887939453125, 0.043853759765625, -0.421142578125, -0.6806640625, -1.328125, 0.572265625, 0.6767578125, 0.7998046875, -1.33203125, -0.13525390625, -0.93505859375, 0.46484375, 0.175048828125, -0.00798797607421875, 0.1697998046875, -0.1075439453125, -0.34033203125, -0.98681640625, -0.1312255859375, 1.6474609375, -0.90576171875, -1.181640625, -0.69970703125, -0.28369140625, -0.2286376953125, -0.86474609375, 1.3349609375, 1.6728515625, -0.2666015625, -0.97216796875, -1.00390625, -0.2213134765625, -0.031890869140625, 0.45068359375, -0.32275390625, -1.4482421875, -0.346435546875, 0.85009765625, 0.288818359375, 0.41455078125, 1.9130859375, 0.07281494140625, -0.8759765625, 0.040679931640625, -0.1375732421875, 0.60888671875, -0.1591796875, 0.0183563232421875, -1.1796875, 0.310302734375, -0.41552734375, -1.65625, 0.340576171875, -0.1148681640625, 0.2353515625, 0.76416015625, -1.8046875, 0.3896484375, 1.2783203125, 0.467041015625, -0.51611328125, 0.9599609375, 0.7548828125, -0.410400390625, -0.69287109375, 0.093505859375, 2.0546875, 0.74267578125, -0.426513671875, 0.184814453125, 1.177734375, -0.59716796875, 0.383056640625, 1.0322265625, 1.3984375, 0.89013671875, -0.246826171875, -0.626953125, -1.064453125, -0.66748046875, 1.4755859375, -0.6669921875, -1.8505859375, 0.3173828125, 0.869140625, 0.1375732421875, -2.48828125, 1.697265625, -0.8193359375, 0.348388671875, 1.0751953125, 0.098876953125, 0.230224609375, 0.268798828125, 0.67724609375, -0.52099609375, 1.2158203125, 1.0478515625, -0.32763671875, -0.10516357421875, -1.3720703125, -0.006412506103515625, 0.353271484375, 0.6591796875, 0.0159759521484375, 0.61474609375, 0.396484375, -0.70263671875, 0.92724609375, 0.1497802734375, 1.30859375, 0.60595703125, 0.53955078125, -0.361083984375, 0.8203125, -1.224609375, -0.2568359375, -1.025390625, 1.4755859375, -0.0745849609375, 0.6513671875, -0.75634765625, -0.53271484375, 0.6171875, 2.4609375, 0.04425048828125, 0.85986328125, 1.404296875, 0.6669921875, -0.12457275390625, -0.5234375, -0.6435546875, 0.8212890625, -1.212890625, 1.0791015625, -1.8955078125, -0.3720703125, 0.90576171875, -0.77392578125, 0.2496337890625, -0.88232421875, -0.39111328125, -0.1380615234375, 0.75, 0.04095458984375, 0.5458984375, -0.2098388671875, -0.9873046875, 0.712890625, -0.587890625, 0.1370849609375, -0.92822265625, -1.119140625, -1.453125, -0.470947265625, -0.2000732421875, 0.380126953125, -0.5185546875, -0.671875, 1.9287109375, -0.372802734375, 0.24658203125, -0.303466796875, -1.34375, -1.1796875, 1.5400390625, 0.2208251953125, -0.42724609375, -1.5859375, 0.58935546875, -0.1278076171875, -0.861328125, 0.654296875, -1.189453125, -1.5322265625, 0.2432861328125, 0.69482421875, 0.94482421875, 0.8662109375, -0.529296875, -2.162109375, 0.1121826171875, 0.89013671875, -0.3408203125, -0.6435546875, 2.365234375, 0.005947113037109375, -1.2685546875, -1.091796875, -1.52734375, -0.238037109375, 0.900390625, 1.525390625, -0.06884765625, -1.4951171875, 1.4169921875, -0.8935546875, -0.395263671875, -0.88671875, -0.498291015625, 0.234619140625, -2.005859375, 0.861328125, 1.701171875, -0.1224365234375, -1.23046875, -0.461181640625, 2.0859375, -0.578125, -1.5244140625, -2.376953125, 0.290771484375, -1.2998046875, -0.65576171875, 0.6328125, -0.61181640625, 0.1993408203125, 2.32421875, -1.1279296875, -0.17919921875, -1.2021484375, -0.2142333984375, 1.13671875, 1.279296875, 0.6904296875, -0.053985595703125, -0.368896484375, -1.24609375, 0.344970703125, -0.73486328125, -0.8095703125, 0.49853515625, -1.4609375, -0.6552734375, -0.50634765625, 2.392578125, 0.36181640625, -0.7431640625, -1.251953125, 0.185546875, -1.62890625, 0.892578125, 0.243896484375, -0.7861328125, 2.1640625, -1.8447265625, -0.241455078125, -0.391845703125, -1.92578125, -1.2861328125, -2.5546875, 0.459228515625, -0.8876953125, 0.0872802734375, 0.363037109375, 0.74853515625, -2.078125, -1.4931640625, 0.06591796875, 0.379638671875, 0.3388671875, 0.290771484375, -0.6396484375, -1.595703125, -1.04296875, -2.09375, -2.78125, 1.37890625, -0.45947265625, 1.7763671875, 0.88525390625, -1.408203125, 0.494384765625, -0.97509765625, -1.779296875, 1.4140625, -1.169921875, 0.6884765625, 0.59814453125, 0.97998046875, -0.09967041015625, -0.43701171875, 0.7255859375, -3.2265625, 0.58349609375, -0.1802978515625, 1.025390625, 0.95947265625, 1.0546875, -0.9267578125, -2.763671875, -0.280517578125, -0.44482421875, 0.08673095703125, 0.6435546875, 1.5517578125, -0.11944580078125, -0.533203125, -0.74365234375, -0.68603515625, -3.1640625, 0.30810546875, 1.7314453125, 0.9833984375, -1.421875, -0.58349609375, -0.5966796875, 1.6474609375, -1.2392578125, 1.5595703125, -1.2314453125, 0.6953125, 2.248046875, -0.53173828125, 0.447265625, 1.671875, 0.41748046875, -0.0350341796875, -0.78076171875, -0.153564453125, -0.448974609375, 0.90576171875, 0.277587890625, -0.8046875, -0.67431640625, 0.97216796875, 0.5205078125, -0.46337890625, -1.11328125, -0.299560546875, -1.1015625, 0.27001953125, 0.224609375, -0.126708984375, 1.4912109375, 0.215087890625, -1.685546875, 0.560546875, 0.8818359375, 0.06231689453125, -1.4765625, 1.27734375, -1.45703125, 0.58251953125, 0.396240234375, 1.837890625, 0.65234375, 0.82861328125, 0.0716552734375, -0.40283203125, -1.5595703125, 0.369384765625, -0.048492431640625, -2.076171875, 0.338134765625, 0.3046875, -0.92724609375, -0.5390625, -1.2705078125, -0.2880859375, -1.8173828125, 0.057586669921875, -0.7119140625, -1.55078125, 0.658203125, -1.2587890625, 0.469482421875, 1.068359375, -0.515625, 0.346435546875, 0.54931640625, -0.9423828125, 0.6767578125, 0.6943359375, -0.5234375, -1.6826171875, -0.8388671875, 0.8837890625, -0.9677734375, 1.322265625, -0.357177734375, 1.41015625, 1.845703125, -0.95361328125, -0.1602783203125, 1.794921875, 0.442626953125, 1.0927734375, 1.109375, -0.67138671875, -0.478271484375, -0.61376953125, 1.7392578125, 0.92236328125, 0.12200927734375, 0.53759765625, -1.205078125, -1.1572265625, -0.845703125, -0.23046875, -1.3271484375, 1.083984375, -1.0263671875, 0.7041015625, 0.09564208984375, 1.84765625, -0.281494140625, 0.9013671875, -0.5283203125, -1.193359375, -0.2646484375, -0.767578125, -0.8095703125, 0.476318359375, 1.470703125, -0.26025390625, 1.091796875, -0.13671875, -0.032806396484375, -0.01058197021484375, -2.048828125, 0.07171630859375, -0.59423828125, -0.0389404296875, 1.1240234375, 0.267333984375, -0.45068359375, -1.7841796875, -0.0770263671875, 1.185546875, 1.080078125, 1.1455078125, -0.82568359375, -1.30078125, 1.3388671875, -0.2186279296875, 0.77685546875, -0.89501953125, -0.57275390625, 0.036651611328125, -0.00875091552734375, -0.22607421875, 0.609375, -0.06268310546875, 0.1961669921875, 0.83642578125, -0.72265625, -0.144287109375, 0.005001068115234375, 0.28466796875, 0.916015625, 1.3525390625, -0.197998046875, -0.529296875, -0.900390625, -0.71240234375, 0.95556640625, -1.0205078125, -0.83447265625, -0.458251953125, -2.2265625, 0.30322265625, -0.24755859375, -0.2125244140625, -1.4306640625, 1.154296875, -1.31640625, -0.755859375, -1.0615234375, -0.62841796875, -0.533203125, 0.84814453125, -0.41015625, -1.140625, 0.02777099609375, -0.2154541015625, 0.6689453125, -1.966796875, 1.44140625, -0.546875, -1.0205078125, -1.6484375, 0.07757568359375, 0.11846923828125, -0.414306640625, 0.03265380859375, -0.28271484375, -0.4140625, 0.845703125, 0.77001953125, -0.1837158203125, 0.208740234375, -3.552734375, 3.21484375, 0.2464599609375, -2.98046875, -0.455322265625, -0.70166015625, -1.2177734375, -0.1746826171875, -0.42529296875, -0.1707763671875, -0.71826171875, 0.8212890625, 0.69970703125, 1.5244140625, 0.09014892578125, -0.053436279296875, -0.55029296875, -0.39208984375, -0.1380615234375, 0.53076171875, 1.3115234375, 0.2320556640625, 0.91748046875, -0.7158203125, -0.12298583984375, -0.494140625, 0.5537109375, -0.77783203125, -0.457763671875, 0.591796875, -0.56201171875, 0.127685546875, -0.373779296875, 0.314208984375, 0.72412109375, 0.6474609375, -0.287109375, -1.970703125, -0.2452392578125, -0.90478515625, -0.303955078125, -0.94287109375, 1.3544921875, -1.4892578125, 1.2744140625, 0.038116455078125, 0.400634765625, -2.466796875, -0.344482421875, 1.302734375, 0.56005859375, -0.5556640625, 1.44140625, -2.5625, -0.6630859375, -0.52978515625, 0.11676025390625, 0.62646484375, -1.6025390625, -0.88671875, -0.06829833984375, 1.224609375, 0.8291015625, 0.71337890625, 0.2216796875, -0.8310546875, 0.48583984375, 0.427001953125, -0.06610107421875, -0.935546875, 0.94287109375, -0.0276947021484375, -1.0224609375, -0.95654296875, 0.214111328125, 0.1427001953125, 0.2108154296875, 1.458984375, 0.0092620849609375, 0.1163330078125, -0.09381103515625, 0.378662109375, 1.1494140625, -0.2449951171875, 1.0478515625, 1.45703125, -1.0986328125, 1.185546875, 1.7626953125, 1.72265625, 0.5830078125, -0.94921875, -0.284912109375, -0.265625, 0.84814453125, 0.08197021484375, 1.05859375, 0.0294036865234375, -0.52783203125, -0.861328125, -0.7451171875, 0.55126953125, -0.449951171875, -0.388671875, -0.40283203125, -0.450927734375, -0.93798828125, -0.6259765625, -1.34375, 1.6748046875, -0.363037109375, -0.312255859375, 0.9248046875, 0.640625, -0.916015625, -1.9443359375, 2.640625, -1.5380859375, 2.345703125, -1.650390625, 0.5400390625, -0.7431640625, -0.230224609375, -0.182373046875, 0.476318359375, 0.77978515625, 0.521484375, 0.422607421875, 1.169921875, 0.0838623046875, -0.546875, -0.65966796875, 0.6396484375, 0.2452392578125, -1.05859375, -1.0048828125, -0.441162109375, 0.0003383159637451172, 2.935546875, 0.9384765625, -0.607421875, 0.414794921875, 0.1087646484375, -0.421142578125, -1.2275390625, 0.443115234375, 0.331787109375, -1.451171875, 0.783203125, 0.4970703125, 0.45263671875, -0.16259765625, 0.78173828125, 0.310302734375, 0.70947265625, 0.17822265625, -0.051849365234375, -0.41357421875, 0.1461181640625, -0.990234375, 0.3447265625, -1.49609375, 0.923828125, -0.1978759765625, -0.357177734375, -0.135009765625, -0.397216796875, -1.365234375, 1.0380859375, -0.272216796875, -0.88671875, -0.6044921875, 1.2255859375, 1.5859375, 0.42822265625, -0.96728515625, 0.62109375, -0.30810546875, -1.5908203125, -1.4677734375, -1.3935546875, 0.1402587890625, 1.0673828125, 0.191162109375, -0.51025390625, -0.10614013671875, -0.09539794921875, -1.521484375, 0.245361328125, -1.8798828125, -0.440185546875, 1.1015625, -1.892578125, 0.12841796875, -0.63134765625, -2.26953125, 0.1873779296875, -1.1044921875, -1.0791015625, -1.75390625, -0.2259521484375, -0.310302734375, 1.3359375, 0.343017578125, 1.3251953125, 1.0556640625, -1.3427734375, 0.2020263671875, 1.380859375, 0.5419921875, 0.1744384765625, -1.4228515625, 1.32421875, 1.2890625, 0.7177734375, -1.322265625, 0.1605224609375, -0.517578125, 1.1376953125, 0.73095703125, 0.452880859375, 1.150390625, 0.751953125, -1.61328125, 1.8076171875, 0.280029296875, -0.98291015625, 1.3779296875, -1.0634765625, -1.3916015625, 0.55908203125, 1.3056640625, 0.0174102783203125, -0.88330078125, 0.72900390625, -0.884765625, 0.1519775390625, 1.0625, 1.154296875, 1.4453125, 1.06640625, -0.728515625, 1.08203125, 1.9228515625, -1.08984375, 1.154296875, -0.765625, -0.3037109375, -0.5517578125, 0.29296875, -0.54150390625, -0.64794921875, -0.0797119140625, 0.5166015625, -0.61474609375, 0.460693359375, 0.155029296875, -0.74365234375, -0.260009765625, 1.181640625, -2.029296875, 0.6865234375, 0.88427734375, 0.9951171875, -1.8701171875, 1.2158203125, 2.576171875, -0.71728515625, 1.3720703125, 0.74072265625, 0.66796875, 0.447021484375, 0.261474609375, -0.392578125, 1.2685546875, -0.250732421875, 0.480712890625, -1.259765625, -0.2469482421875, -0.15283203125, 1.2626953125, 0.78564453125, 0.39208984375, -1.55859375, 0.370361328125, -0.576171875, -0.818359375, -0.2431640625, -0.72021484375, 0.6328125, 0.59375, 0.1031494140625, -1.3857421875, -0.52001953125, -0.286376953125, -0.81787109375, -1.7763671875, 1.4921875, 0.572265625, 1.0263671875, -0.1263427734375, -0.677734375, -0.2274169921875, 1.01171875, 2.04296875, -0.27099609375, -0.050079345703125, 2.12109375, 0.27978515625, 0.50634765625, -0.544921875, -1.1103515625, 0.70654296875, -2.392578125, -0.309326171875, 2.04296875, -0.18212890625, -0.2020263671875, 0.6083984375, 1.8369140625, -0.7412109375, -0.49609375, -0.1531982421875, 0.92626953125, 1.3681640625, -1.2822265625, -1.1796875, 1.572265625, -0.5498046875, -1.3935546875, -0.164306640625, 0.9150390625, -0.28662109375, 0.408203125, -0.59814453125, 0.46240234375, 1.04296875, 0.0782470703125, -0.724609375, 0.60693359375, -0.469482421875, -1.6435546875, 1.572265625, -2.794921875, 1.265625, 1.8515625, 1.294921875, 1.892578125, -0.479248046875, 1.0791015625, -1.052734375, -1.7705078125, -0.55908203125, -0.72216796875, 1.1015625, 0.919921875, -3.185546875, 1.1669921875, -0.8193359375, 1.3095703125, 0.010986328125, 0.91552734375, -1.197265625, -0.341796875, -0.4443359375, 0.061767578125, 0.7412109375, 1.326171875, 0.32568359375, 0.57568359375, 0.66064453125, 0.10418701171875, 0.360107421875, 1.4873046875, -0.275390625, -1.9873046875, 0.8076171875, -0.8681640625, 0.31396484375, 0.420166015625, -1.0595703125, -0.1611328125, 0.0333251953125, 1.5283203125, 1.564453125, 1.880859375, -0.1685791015625, 0.6220703125, -0.28662109375, 1.1953125, 0.4228515625, 0.419189453125, -0.62451171875, 0.74560546875, -0.71923828125, -0.8955078125, 0.225830078125, -1.0703125, -0.77001953125, 2.205078125, -0.84130859375, 0.06536865234375, 0.5625, -0.77392578125, 0.533203125, -0.3818359375, 0.90185546875, 0.99755859375, 0.10809326171875, -0.69580078125, 0.0081787109375, -0.9501953125, -1.9453125, -1.7197265625, -2.7265625, -0.1656494140625, 0.38818359375, 1.4365234375, -0.443603515625, 0.544921875, -0.78759765625, -0.48779296875, 0.0919189453125, -0.6259765625, -0.56103515625, -0.145751953125, 0.7109375, 1.404296875, -0.96630859375, 0.149169921875, -0.258544921875, 0.1907958984375, -0.25830078125, -0.6611328125, -0.1949462890625, 0.708984375, -0.5810546875, -0.408447265625, 2.41796875, -0.78955078125, 0.020965576171875, -0.265380859375, -1.4765625, -1.4677734375, -0.54833984375, -1.5712890625, 1.6796875, 0.3896484375, 1.580078125, -0.97900390625, -0.041290283203125, 0.239013671875, -0.361572265625, 0.7353515625, 0.1368408203125, -0.74169921875, 0.099853515625, -0.87109375, -0.01534271240234375, -0.3115234375, -0.787109375, -0.05755615234375, 0.55126953125, 1.048828125, 0.1856689453125, -0.3349609375, -0.1944580078125, -1.654296875, 0.91943359375, -0.2193603515625, 1.294921875, 0.419677734375, 0.48779296875, 0.81982421875, 0.179443359375, 0.1890869140625, 2.271484375, 1.2470703125, -0.364501953125, -0.0117645263671875, -0.4375, 0.86865234375, -1.107421875, -0.9560546875, 0.9814453125, -0.05084228515625, -0.306396484375, -0.280029296875, -0.34912109375, -0.9697265625, 0.85546875, 1.4609375, 0.81640625, 0.463134765625, -0.73583984375, 1.2548828125, 0.57861328125, 1.0498046875, -1.423828125, -0.7001953125, 0.05316162109375, -0.21044921875, 0.2998046875, -1.5166015625, -1.818359375, 0.323486328125, -0.54443359375, -0.5703125, -2.357421875, 0.06597900390625, 0.9462890625, 0.450439453125, 0.69775390625, -0.399169921875, -0.58642578125, 0.10394287109375, -0.880859375, 0.1790771484375, 0.67529296875, 1.396484375, 1.0107421875, -1.7568359375, -0.37939453125, -1.5927734375, -0.7802734375, 0.254150390625, -0.10357666015625, -1.09765625, -0.019378662109375, 1.060546875, -0.3994140625, 0.7099609375, 1.1884765625, -0.7001953125, 0.286376953125, 1.7314453125, -2.736328125, -1.861328125, 1.0302734375, 1.49609375, -0.8056640625, -0.035064697265625, 0.52001953125, 0.97412109375, -0.77490234375, -0.658203125, 0.3955078125, -0.051544189453125, 0.7080078125, -0.188232421875, -1.7666015625, -0.50927734375, 0.03070068359375, 0.57861328125, -0.76904296875, 1.513671875, 0.759765625, -0.311767578125, -1.111328125, -0.43896484375, -0.498779296875, -1.1572265625, 2.625, -1.4404296875, -0.10076904296875, 0.87060546875, -0.159423828125, -1.212890625, 0.246337890625, 1.822265625, 1.21875, -0.311279296875, 0.50244140625, -0.8974609375, -1.9921875, -0.1649169921875, -1.2861328125, 0.07025146484375, -0.050872802734375, 0.8720703125, 0.94140625, 1.1533203125, -0.285400390625, 0.2183837890625, -0.392578125, -1.4609375, 0.67138671875, 0.85498046875, -0.053192138671875, -0.2327880859375, -0.2064208984375, 2.025390625, 0.31787109375, 0.383056640625, -0.363525390625, -0.1876220703125, -1.361328125, -2.447265625, -1.0078125, 0.22509765625, 1.916015625, -0.32080078125, 2.771484375, -0.2666015625, 0.2481689453125, 1.16796875, -2.3828125, 0.0079345703125, -0.890625, 0.09088134765625, -0.71142578125, -0.453857421875, -1.6669921875, 0.54931640625, -1.44140625, -1.439453125, 0.7998046875, -0.2958984375, -1.3349609375, 0.14208984375, -0.320556640625, -1.9794921875, -0.468994140625, 0.48779296875, -0.07208251953125, -0.48193359375, 0.875, 0.48779296875, 0.716796875, -0.02783203125, -0.6513671875, -0.2379150390625, 1.658203125, -0.03289794921875, 0.12139892578125, -1.041015625, 1.0712890625, -1.2451171875, -1.94140625, 0.6220703125, -0.101318359375, 0.59326171875, 1.1220703125, -0.1666259765625, -0.2325439453125, 0.1883544921875, 0.6728515625, 0.89453125, 0.484375, -1.55078125, 0.08978271484375, 0.28955078125, 0.6171875, -1.01171875, 0.053924560546875, 1.7216796875, -0.419189453125, 0.269775390625, 0.1849365234375, 0.312744140625, 1.5546875, 0.360595703125, 0.6171875, 0.98779296875, 0.485107421875, -0.3798828125, -1.5576171875, 0.5771484375, 1.57421875, -0.069091796875, 0.09661865234375, -2.671875, -1.90625, 1.3076171875, 0.1705322265625, 0.357177734375, 1.6083984375, 1.6884765625, -1.34375, 0.912109375, -1.818359375, -0.5029296875, -1.4990234375, 0.73486328125, 0.63818359375, -1.2919921875, -0.52197265625, 1.775390625, 0.0924072265625, -1.0625, 1.3818359375, 1.001953125, 0.89599609375, 1.2109375, -0.162109375, 1.27734375, -1.4482421875, 0.340087890625, -0.2333984375, 0.08514404296875, -0.369873046875, -0.29736328125, -1.4365234375, 0.55126953125, -1.7900390625, 1.322265625, 0.352783203125, 1.3251953125, -2.205078125, 0.10791015625, 0.00603485107421875, -0.0076751708984375, -0.900390625, -0.7197265625, -0.282958984375, -1.6875, 1.2412109375, -0.689453125, 0.8720703125, 1.203125, -0.6630859375, -0.630859375, 0.59130859375, 1.5263671875, 0.50244140625, -0.984375, -0.326904296875, 1.8525390625, -0.151123046875, -1.0810546875, 1.3359375, -0.1705322265625, 0.144287109375, -0.08392333984375, 0.90478515625, 0.728515625, -0.49072265625, -0.289306640625, 0.356689453125, -0.0711669921875, 0.0121612548828125, -0.299072265625, 1.7626953125, -1.33203125, -0.164306640625, 0.65771484375, -0.3193359375, -1.7685546875, 0.1370849609375, -0.209716796875, -0.15283203125, -0.358154296875, -0.1903076171875, 0.46630859375, 0.03900146484375, -1.3349609375, 0.111083984375, 0.06671142578125, 0.229248046875, 1.4873046875, -0.499755859375, 1.0205078125, 1.6162109375, -0.3525390625, -0.2998046875, -0.81396484375, 1.5615234375, 0.53515625, 0.5703125, 2.16796875, 0.3525390625, -0.11181640625, 0.90380859375, 0.900390625, -0.2347412109375, 1.234375, 0.9130859375, 0.92529296875, 1.806640625, 1.9033203125, -1.1171875, 0.2484130859375, -0.2042236328125, 0.43994140625, 0.362060546875, 0.030242919921875, -1.169921875, 1.513671875, -0.96484375, 0.459228515625, 0.004459381103515625, -2.59765625, 0.73876953125, 2.298828125, 0.90625, 0.324951171875, 1.4326171875, -0.5341796875, 0.2529296875, -1.3173828125, 0.47607421875, 0.493408203125, 2.384765625, -0.81103515625, -0.44873046875, 0.76025390625, 1.359375, 1.76953125, 1.025390625, -0.0899658203125, 1.396484375, 1.0634765625, 0.394775390625, 2.1875, -0.143310546875, -1.830078125, -1.2705078125, -0.1839599609375, 1.271484375, 0.78759765625, -1.365234375, 0.03045654296875, -0.059814453125, -2.48828125, -0.228271484375, 0.286865234375, 0.70166015625, -0.515625, -0.60791015625, 0.4921875, 0.491943359375, -0.7001953125, -0.352783203125, -2.3543834686279297e-05, 0.85302734375, -0.91162109375, -0.221435546875, -0.1876220703125, 1.19921875, 0.8076171875, -0.78955078125, -0.3876953125, -2.31640625, -0.63232421875, -0.454345703125, 0.51806640625, 2.390625, 0.6591796875, 1.078125, -0.19580078125, -0.31884765625, 0.8583984375, 1.404296875, 1.0908203125, 0.448486328125, 0.081298828125, 0.289306640625, -0.203125, -0.9912109375, -0.059844970703125, 0.310546875, -0.9599609375, 0.54638671875, 1.494140625, 0.7802734375, -0.455322265625, 1.890625, 0.5341796875, 0.1612548828125, 0.1441650390625, -2.806640625, 1.107421875, -0.96826171875, -0.448486328125, -0.59326171875, 1.5458984375, 0.5341796875, 1.3486328125, 0.1900634765625, 0.70556640625, 0.67724609375, 2.3359375, 0.387451171875, -0.93115234375, 0.73388671875, 0.492919921875, -1.4326171875, -0.323974609375, -1.6982421875, 0.2449951171875, -1.1708984375, -0.6962890625, -0.06341552734375, 0.9267578125, 0.62939453125, -0.7861328125, 0.26123046875, 0.7978515625, -1.05078125, 0.69140625, 0.93798828125, -0.88232421875, 0.1317138671875, -0.78076171875, 0.0082550048828125, -0.2152099609375, -0.66259765625, -1.1201171875, 0.6142578125, 1.30078125, -0.96142578125, 0.521484375, 0.421875, -0.1435546875, -0.19482421875, -0.5068359375, 1.2744140625, 0.301513671875, -0.9814453125, -0.31298828125, -0.302978515625, 1.1962890625, 0.00859832763671875, 0.599609375, -0.52978515625, 0.261962890625, 0.73681640625, -0.316162109375, 1.4375, 1.4443359375, -0.7119140625, 1.734375, 0.54541015625, 0.2061767578125, 0.060791015625, -1.05078125, -0.483642578125, 0.56298828125, -0.220703125, 0.75390625, 0.168212890625, -1.2060546875, 0.9384765625, -0.11187744140625, -1.439453125, 0.73486328125, 2.009765625, 0.1844482421875, 0.8115234375, -1.9091796875, 1.033203125, 0.412353515625, -0.314453125, -0.56396484375, -0.6767578125, -0.86962890625, -0.9189453125, -0.01334381103515625, -0.837890625, -1.2333984375, 1.0595703125, -0.42626953125, -0.17236328125, -0.417724609375, -1.916015625, 0.41650390625, -0.036956787109375, -0.452392578125, 1.443359375, -0.20751953125, -0.47314453125, -1.42578125, 1.0546875, -1.3544921875, 0.90966796875, 2.29296875, -0.1121826171875, 1.486328125, 1.3505859375, -0.433837890625, -1.0078125, -1.3154296875, -0.32275390625, -0.89013671875, 0.39208984375, 0.29541015625, -2.126953125, -0.0167083740234375, 0.335205078125, 2.525390625, -0.89794921875, -0.6708984375, -2.31640625, -1.447265625, 1.0048828125, 2.41015625, 0.99169921875, -0.461669921875, -0.204833984375, 1.77734375, 0.48193359375, -0.76318359375, -0.1484375, -0.89892578125, 0.11334228515625, 0.117431640625, -0.966796875, 0.15673828125, 1.3974609375, -0.50244140625, -1.212890625, 0.482666015625, -0.41357421875, 0.284912109375, 0.79931640625, 1.8935546875, 1.634765625, 1.693359375, 1.58984375, -0.10614013671875, -0.6220703125, 0.064453125, -0.9404296875, 2.265625, -1.7734375, -2.451171875, 0.54345703125, -0.1778564453125, -0.50537109375, 0.15478515625, -1.38671875, -1.4326171875, -0.8515625, -0.92822265625, -2.12890625, -0.82470703125, 0.63525390625, -2.380859375, 0.09698486328125, 0.9189453125, -0.390380859375, 0.64111328125, 0.239501953125, 0.98876953125, -0.1055908203125, 0.295166015625, -0.7998046875, -0.728515625, -0.30419921875, -0.94677734375, 1.2001953125, 0.71435546875, -0.96875, -1.2685546875, -0.434814453125, -1.1845703125, 0.469482421875, -1.076171875, -0.0499267578125, -0.69189453125, -0.29248046875, 0.78564453125, 0.8037109375, -1.2216796875, 0.036285400390625, 0.9609375, 1.69921875, 1.5966796875, -0.71240234375, 0.81005859375, -0.186279296875, -1.6494140625, -1.3271484375, 1.75390625, 1.0087890625, -2.005859375, -1.595703125, -0.60986328125, 0.0231781005859375, -1.6044921875, -0.69140625, -0.611328125, -0.125244140625, 0.900390625, -0.47509765625, -1.7568359375, -0.366455078125, -0.2152099609375, 0.1285400390625, -0.11505126953125, 0.16796875, 0.5673828125, 1.1748046875, -0.0718994140625, 0.71728515625, -0.82568359375, -0.355224609375, 0.401123046875, 0.87060546875, -0.60400390625, 0.1480712890625, -0.1495361328125, 0.2064208984375, 0.2412109375, 1.01171875, 0.57080078125, 0.931640625, 1.220703125, -1.166015625, 0.591796875, 0.17138671875, -0.603515625, -0.677734375, 0.9833984375, -0.276611328125, -0.315673828125, 0.13134765625, -1.224609375, 0.387451171875, -1.2958984375, -0.80517578125, 0.375732421875, -1.04296875, -0.57666015625, -1.12109375, -1.3017578125, -0.386474609375, -0.26513671875, -0.08905029296875, -0.07232666015625, -0.3876953125, -2.16015625, 0.7890625, -0.2252197265625, -0.281005859375, 1.0234375, 1.044921875, 0.6826171875, 0.035400390625, 0.2322998046875, 1.841796875, 0.763671875, -1.0009765625, 1.140625, 0.0897216796875, -0.280029296875, 0.84423828125, 0.64990234375, 0.69140625, 1.4052734375, -2.25390625, 0.86376953125, 0.19775390625, -0.7294921875, -0.234619140625, 0.4404296875, -0.28466796875, 1.919921875, -1.06640625, -1.2001953125, 0.0090789794921875, -1.5947265625, 1.8447265625, 0.133056640625, 1.1875, -0.736328125, -0.1566162109375, -1.4599609375, -1.0087890625, 0.1868896484375, 1.2451171875, -0.80029296875, -1.068359375, -0.5634765625, 1.916015625, 0.88818359375, -0.9873046875, -0.056884765625, -1.4208984375, -0.64599609375, 1.3115234375, 1.0498046875, 0.057891845703125, -0.130615234375, -1.1962890625, 0.75830078125, -0.9345703125, -0.53125, -1.634765625, -0.35546875, -0.013336181640625, -1.271484375, -0.01328277587890625, 1.177734375, 0.345703125, -1.3154296875, -0.82763671875, 0.61669921875, -1.060546875, 1.3310546875, -1.6396484375, 0.2076416015625, -0.18359375, -0.30810546875, -0.347900390625, -0.37939453125, 2.09375, -0.5205078125, -0.53466796875, -1.0576171875, 2.47265625, 0.1641845703125, -0.0635986328125, -0.243408203125, -0.484619140625, -0.1541748046875, -0.732421875, 0.73681640625, -1.7021484375, -0.6083984375, -1.3828125, 1.2724609375, 0.51513671875, 0.181640625, 0.8076171875, -0.435302734375, 0.92236328125, -0.8212890625, 1.0244140625, 0.19189453125, 0.8984375, -1.3056640625, 0.5244140625, 0.66064453125, -1.3857421875, 0.411865234375, 0.2548828125, 0.378173828125, 0.33837890625, -0.471435546875, -0.05523681640625, -0.050811767578125, -1.7841796875, -1.1552734375, 0.498779296875, 0.50634765625, -0.35888671875, 0.41943359375, 0.452880859375, 0.8095703125, -0.6875, -1.216796875, 0.147216796875, -0.318359375, -0.1456298828125, -1.3359375, -0.54443359375, -0.4638671875, -1.5830078125, -0.66455078125, 1.3056640625, 1.0927734375, 1.0009765625, -1.197265625, -0.1934814453125, 1.06640625, 0.0231475830078125, -0.9287109375, 0.81298828125, -3.359375, 0.2261962890625, 1.0595703125, -0.383056640625, 1.646484375, 0.900390625, 0.4580078125, 0.87158203125, 0.07855224609375, -0.478515625, 1.453125, -0.75732421875, -1.52734375, -2.00390625, -1.1171875, -0.494384765625, 0.4033203125, -0.356689453125, 0.01357269287109375, -1.2607421875, 0.10223388671875, -0.40283203125, -0.62109375, 1.0654296875, -1.337890625, -0.88916015625, 2.4609375, 0.5302734375, 1.388671875, 0.29638671875, 0.1949462890625, -1.3115234375, -0.12274169921875, -1.15234375, -0.075439453125, -0.218994140625, 1.6865234375, -0.31396484375, -1.568359375, 0.55712890625, -0.13916015625, 0.6787109375, 1.240234375, 0.464111328125, -1.1435546875, -0.1468505859375, 1.7138671875, -1.5859375, 0.465576171875, 0.239501953125, -0.53125, 0.9140625, -1.7275390625, -0.2342529296875, -1.00390625, -0.65380859375, 0.0902099609375, -1.6123046875, 0.89111328125, -1.642578125, -1.466796875, -0.70654296875, -0.8740234375, 0.298583984375, 1.7080078125, -1.4375, -1.2939453125, 0.68408203125, 1.3310546875, -1.8486328125, 1.1376953125, -0.1534423828125, 1.1484375, -0.2130126953125, 0.08966064453125, 0.513671875, 0.46728515625, -0.2491455078125, 0.022796630859375, 0.006786346435546875, -0.947265625, 0.56640625, 1.033203125, 1.8271484375, 1.9423828125, 0.8642578125, 0.556640625, -0.3447265625, 0.7763671875, 0.1697998046875, 0.0989990234375, -1.2587890625, -0.355712890625, 0.2218017578125, -1.1162109375, -0.662109375, -0.19384765625, 0.7724609375, -0.6923828125, -0.89453125, -0.07501220703125, -0.3671875, 0.708984375, 0.82861328125, -1.1494140625, -0.626953125, -1.5478515625, 0.541015625, -0.4345703125, 0.1456298828125, -0.104736328125, -1.1376953125, 1.8916015625, 0.4638671875, -0.46142578125, 1.7763671875, -0.6572265625, -0.56103515625, 1.0478515625, -0.1798095703125, -0.58984375, -0.47802734375, -0.06805419921875, 0.75, 0.8076171875, -0.6943359375, -1.1416015625, -0.98681640625, -0.2254638671875, 0.7451171875, 1.2705078125, 0.8837890625, 1.0966796875, 1.251953125, -0.339111328125, -0.66162109375, 0.53369140625, -1.244140625, -0.65380859375, 0.9970703125, 1.4912109375, -0.873046875, -1.671875, -0.85791015625, -0.9501953125, 1.3818359375, 2.2578125, 0.13134765625, -0.77294921875, 1.2705078125, -0.4892578125, 1.72265625, 0.77392578125, 0.498046875, 1.2646484375, -1.7255859375, -0.51513671875, -2.08203125, 0.37109375, -1.208984375, -0.385498046875, -0.3837890625, -0.88818359375, -0.83935546875, 0.5595703125, -0.7568359375, 0.06298828125, 1.01171875, -1.7431640625, -0.28369140625, -0.66650390625, -0.022369384765625, 0.265625, -1.025390625, -2.345703125, 0.09814453125, 0.970703125, 0.1903076171875, -0.85986328125, -0.32177734375, -0.228759765625, -2.33203125, -0.0635986328125, -1.0390625, 1.4208984375, -0.259765625, -0.11859130859375, 1.314453125, 0.81982421875, -0.56982421875, 0.5107421875, -0.8798828125, -0.03729248046875, -0.379638671875, -0.521484375, 1.5791015625, 1.0009765625, 0.2734375, 1.3134765625, 0.138916015625, 1.197265625, -0.1470947265625, -0.2193603515625, -1.1162109375, -0.0222015380859375, 0.0184478759765625, 0.286376953125, 0.468017578125, -0.67822265625, 0.2086181640625, -0.60546875, -0.351806640625, -1.0537109375, 0.459228515625, 0.95947265625, 0.958984375, -1.0478515625, -0.578125, -0.5166015625, 1.4326171875, 1.1474609375, -0.65234375, 1.8037109375, -0.76318359375, -0.76806640625, 0.7607421875, -0.2493896484375, 1.0517578125, -0.73095703125, -0.2235107421875, -1.4169921875, -0.35986328125, -0.54443359375, -1.1689453125, 0.576171875, -0.1392822265625, 0.489013671875, -1.07421875, 0.69287109375, 0.55419921875, 1.271484375, 0.3046875, -1.833984375, -0.2420654296875, -1.1728515625, 1.19921875, -0.82470703125, 0.43212890625, -0.19580078125, 2.80078125, 0.1959228515625, -0.177001953125, 0.361572265625, 0.671875, -1.4931640625, -0.06439208984375, -1.4033203125, -1.9248046875, 0.59912109375, 0.833984375, 0.931640625, 0.609375, -0.74072265625, 0.57763671875, -0.375244140625, -0.78271484375, -1.0869140625, -0.314208984375, -1.2626953125, -0.20751953125, 0.59912109375, -0.87060546875, 0.7998046875, -1.20703125, 0.299072265625, -1.474609375, -1.4951171875, 0.474365234375, 1.1455078125, 0.0751953125, 0.61376953125, -0.74560546875, 1.12890625, 0.26220703125, -0.78662109375, 1.2705078125, 0.05059814453125, -0.76904296875, -0.7119140625, -0.1788330078125, 0.480224609375, -0.95263671875, -0.0897216796875, 0.9052734375, -1.947265625, 0.86376953125, 0.0543212890625, 0.6376953125, -0.0784912109375, -0.802734375, 0.3564453125, 0.5029296875, 0.2310791015625, 0.771484375, 0.99072265625, 0.6005859375, -0.0039005279541015625, -0.9072265625, -1.8896484375, 1.0615234375, -0.52294921875, -0.07843017578125, -1.693359375, 0.445556640625, -0.359619140625, 0.90673828125, 0.1229248046875, -0.6767578125, 1.1337890625, 0.009063720703125, 0.452880859375, 0.31982421875, 1.0771484375, -1.7001953125, 1.232421875, -0.8603515625, -1.3564453125, 0.293701171875, 0.83154296875, 0.3623046875, -0.091796875, -0.361328125, -2.345703125, -1.7919921875, -0.185302734375, 1.2900390625, 2.19921875, 0.94873046875, -0.034576416015625, 0.615234375, 0.1966552734375, 0.82080078125, -0.66064453125, -1.16015625, -1.4150390625, -1.4345703125, -1.90234375, 0.11767578125, -0.29833984375, -0.2305908203125, -0.2301025390625, -0.982421875, 0.116943359375, 1.390625, -0.87744140625, -0.52685546875, 0.070556640625, 0.38720703125, 0.63134765625, 0.52490234375, -0.14111328125, 1.3173828125, 0.64208984375, 0.9287109375, -0.2431640625, -0.76318359375, -2.46875, 0.290771484375, 0.45068359375, 0.9599609375, 0.79296875, -0.7080078125, 0.44189453125, -1.27734375, -0.1881103515625, -1.3916015625, -0.1964111328125, -0.7412109375, -0.8603515625, -1.8759765625, -1.5908203125, -0.48291015625, -0.5771484375, -0.6845703125, 1.7587890625, -0.431640625, -0.69384765625, -0.7763671875, 1.30859375, -0.765625, -0.09320068359375, -1.8603515625, -0.09783935546875, 1.474609375, 0.26953125, 0.94970703125, 0.892578125, 2.75, -0.50244140625, 0.794921875, 1.396484375, -0.07977294921875, 0.0721435546875, 0.82373046875, -1.1650390625, -0.6513671875, 0.75244140625, 1.0478515625, 0.8291015625, -0.1705322265625, 1.8173828125, -0.86376953125, -2.0390625, -0.07769775390625, 0.72412109375, 1.37109375, 1.07421875, 1.4892578125, -0.9248046875, -2.546875, 0.0447998046875, 1.7763671875, 0.95068359375, 1.0849609375, 0.0118865966796875, 1.2392578125, -0.74169921875, -0.55859375, -0.98583984375, 0.814453125, 0.1632080078125, -0.375244140625, 1.283203125, 0.153564453125, -0.7939453125, 0.71044921875, 0.02606201171875, 0.76318359375, -0.53173828125, -0.255859375, -0.49755859375, 1.5146484375, 0.724609375, -2.2421875, 1.158203125, -0.53271484375, -0.701171875, 1.07421875, -1.37890625, -0.92041015625, 1.099609375, -2.16796875, -0.07135009765625, -0.41845703125, -0.47021484375, -0.429931640625, 0.42919921875, -0.75048828125, 0.2626953125, 0.091064453125, 0.42724609375, 1.31640625, 0.6474609375, -1.560546875, -0.0173492431640625, -2.177734375, 1.8681640625, -0.10577392578125, -1.65234375, -0.55615234375, 1.16015625, 0.51171875, -0.331298828125, 0.583984375, -1.7666015625, -1.0283203125, -1.6083984375, 1.2275390625, 0.5830078125, 2.6171875, -0.705078125, -0.354248046875, -0.53125, -0.30859375, -0.8056640625, 0.09149169921875, 1.255859375, -1.4052734375, -0.84814453125, -0.96923828125, -1.9521484375, 0.1763916015625, -1.740234375, -0.70361328125, 1.1181640625, -1.3251953125, -0.428955078125, -0.93603515625, -0.2174072265625, -1.591796875, 1.3154296875, -0.90966796875, -0.26220703125, -0.12841796875, -0.82958984375, 0.7529296875, -0.049560546875, 0.59228515625, 1.3203125, -1.20703125, 2.068359375, 0.7880859375, 1.2919921875, 0.1385498046875, -2.419921875, 0.423828125, 1.10546875, -0.5986328125, -0.07940673828125, -0.4228515625, 2.4453125, -1.4033203125, 0.91455078125, -0.0254058837890625, -0.71484375, -0.310302734375, 0.5048828125, 0.88134765625, -0.1463623046875, 0.93017578125, -0.2000732421875, -0.034210205078125, -0.67236328125, 1.1396484375, -0.468017578125, 1.3994140625, -1.0224609375, 0.40869140625, -0.0156707763671875, 1.1318359375, 0.452392578125, 0.06500244140625, -0.75537109375, 0.67919921875, 0.94287109375, 0.5078125, 0.413330078125, -0.15234375, -2.48828125, -2.0390625, 1.6630859375, 0.21630859375, -1.5166015625, -0.96875, -0.77392578125, -0.54296875, 0.041229248046875, -0.63916015625, 0.2261962890625, -0.888671875, -0.25732421875, 0.054840087890625, 0.505859375, 0.732421875, 0.36474609375, 0.970703125, -1.421875, 0.88525390625, 0.658203125, -0.59814453125, -1.4599609375, 1.302734375, -0.416015625, -0.41748046875, -0.5986328125, -0.4462890625, -0.1258544921875, 0.98779296875, -1.1328125, -0.90673828125, 1.431640625, -0.372802734375, -0.77978515625, 0.96728515625, -2.69921875, -0.166259765625, -1.2412109375, -0.76953125, 1.580078125, 1.2236328125, 0.062469482421875, -0.66259765625, -0.53466796875, -1.443359375, 1.916015625, 0.287841796875, -0.12164306640625, 2.11328125, 2.220703125, 1.43359375, 0.99853515625, 0.892578125, 0.20947265625, 0.429931640625, 1.279296875, -0.08111572265625, -1.2705078125, 2.28125, -0.60986328125, 2.314453125, 0.29345703125, 0.16455078125, 0.97119140625, -0.2861328125, 0.10772705078125, 0.7822265625, 1.11328125, 0.495361328125, -0.7470703125, -0.0919189453125, -0.216796875, -0.359619140625, -0.404296875, -0.904296875, 0.061370849609375, -0.1422119140625, -2.18359375, 0.78955078125, -0.63623046875, 0.51513671875, 0.11328125, 1.6845703125, 1.2470703125, 0.313232421875, 3.041015625, 0.99072265625, 1.849609375, 0.09716796875, 0.2049560546875, 1.1259765625, -1.853515625, -0.09454345703125, 0.90185546875, 0.890625, 0.38818359375, -0.08404541015625, 2.064453125, -0.2139892578125, -1.0537109375, 0.876953125, -0.861328125, 1.6787109375, -0.472412109375, -0.72705078125, -0.09149169921875, 1.947265625, -2.349609375, 0.027557373046875, -0.06854248046875, 1.490234375, -0.537109375, -0.65771484375, 0.74169921875, 0.295166015625, 0.240478515625, -1.0810546875, 0.56201171875, 0.57421875, -0.83056640625, 1.8974609375, -1.009765625, -1.0380859375, 0.79638671875, 1.3974609375, 0.1319580078125, -1.1875, -0.257080078125, -0.8046875, 0.2142333984375, 0.294677734375, 0.07513427734375, 0.85400390625, 0.468017578125, 0.85400390625, 1.7353515625, 0.367919921875, -0.79638671875, 0.716796875, 0.1400146484375, -0.481689453125, -0.1568603515625, -1.14453125, -0.75, 0.6494140625, 0.701171875, -0.5771484375, -0.293212890625, 1.521484375, -0.400634765625, 0.049530029296875, -0.06903076171875, 0.3798828125, 0.37744140625, -0.132568359375, 1.685546875, 0.01093292236328125, -2.201171875, -0.990234375, -1.8583984375, 1.7802734375, -1.5625, -1.732421875, -1.90625, -1.7763671875, -0.880859375, 2.380859375, -0.4833984375, -1.8505859375, -0.1456298828125, -0.49755859375, -1.6376953125, 0.87109375, 0.470703125, 0.92626953125, 1.70703125, -1.5771484375, 0.68701171875, 0.45263671875, 0.775390625, -0.8310546875, 0.294189453125, -0.5087890625, 0.82568359375, 0.5634765625, -1.232421875, 0.59033203125, -1.189453125, -0.76416015625, 0.06744384765625, -0.349609375, 0.41943359375, -0.31396484375, -0.2568359375, -0.322509765625, -1.1474609375, -1.0810546875, 1.3173828125, -0.080810546875, -0.39501953125, -0.151123046875, -0.37060546875, -1.66796875, -0.7861328125, -0.57861328125, -0.80859375, 1.34375, -1.0966796875, -0.6708984375, 1.669921875, -0.002490997314453125, -0.265625, 0.64404296875, 0.0023975372314453125, 0.019927978515625, 1.0576171875, -0.92626953125, -1.4765625, 0.90771484375, -1.529296875, -0.98193359375, 1.18359375, -0.912109375, -0.144775390625, -0.2376708984375, -0.362060546875, 1.712890625, 0.44140625, -1.064453125, -0.79248046875, 0.3466796875, 0.400390625, 1.828125, -1.333984375, -0.57470703125, -0.697265625, 0.302490234375, -0.08343505859375, 2.125, -0.346435546875, 0.56005859375, -0.284912109375, 3.81640625, 0.0024471282958984375, -0.15625, 0.2471923828125, 0.388916015625, -0.159912109375, 1.076171875, 0.454833984375, -0.6650390625, -0.47900390625, -1.3408203125, 0.583984375, 0.0863037109375, -0.2337646484375, 1.0498046875, 0.7138671875, -0.1156005859375, -0.8046875, 0.44482421875, -2.11328125, 1.1337890625, 0.1522216796875, 1.0576171875, 0.3984375, 1.5322265625, 0.53466796875, 0.61328125, 0.2191162109375, 0.12408447265625, 0.2264404296875, -0.09368896484375, -0.6962890625, -0.54296875, 1.5810546875, 2.5, 0.501953125, 2.392578125, 1.92578125, 1.5869140625, -0.10052490234375, -1.9716796875, -0.375244140625, 0.50537109375, -0.437744140625, 0.437255859375, -0.382568359375, 0.03704833984375, -0.368408203125, 2.44921875, 1.2451171875, 0.1463623046875, -1.6728515625, 0.41259765625, -0.27490234375, -0.414306640625, -0.82275390625, -0.91552734375, -1.1259765625, 0.826171875, 0.7958984375, -1.5205078125, -0.0190887451171875, -0.70703125, -0.3359375, -0.97705078125, 0.51318359375, -1.087890625, 0.97412109375, -0.51904296875, -1.08984375, -0.00251007080078125, 0.2318115234375, -0.174072265625, 0.40283203125, 0.9736328125, 2.14453125, 0.69970703125, -0.583984375, 0.54541015625, 0.94189453125, -1.0439453125, -1.3740234375, 0.58349609375, 1.0498046875, -0.45166015625, 0.01528167724609375, 0.2489013671875, 0.419189453125, 0.662109375, -0.76025390625, 1.017578125, 0.0251617431640625, 0.12890625, 0.403076171875, -0.093017578125, 0.0977783203125, 1.638671875, 1.6474609375, -0.113037109375, 0.0811767578125, 0.80517578125, 0.91357421875, 0.65185546875, 0.38623046875, -0.78662109375, 1.7236328125, 1.44140625, 1.265625, -0.78564453125, -0.2001953125, 1.369140625, -0.223876953125, -0.91064453125, -0.2890625, -1.1884765625, 0.3330078125, -3.4453125, -0.5419921875, 1.423828125, -2.068359375, -1.0390625, -0.08245849609375, 1.1064453125, -0.66943359375, 1.1640625, -1.19140625, -1.0107421875, 0.51318359375, 0.63330078125, -0.100341796875, 0.0034694671630859375, -1.1103515625, -0.62109375, 0.93408203125, 0.396728515625, 1.57421875, -0.5205078125, 1.9833984375, -0.2880859375, -0.583984375, 0.09893798828125, 0.73974609375, -0.5576171875, 0.1942138671875, -0.78369140625, -0.90673828125, 2.306640625, -0.5390625, -0.85986328125, -0.822265625, -1.587890625, -1.5712890625, 1.072265625, 0.59326171875, -0.353271484375, -0.0797119140625, 0.7314453125, 1.3720703125, 0.6279296875, -0.1800537109375, -2.486328125, 0.60986328125, -1.1123046875, 0.8828125, 1.427734375, 1.0712890625, -1.0673828125, -0.50390625, 0.034454345703125, -0.515625, 0.7236328125, 0.056182861328125, -1.263671875, 0.5205078125, -0.8046875, 0.39306640625, 1.0126953125, 0.295166015625, 0.77392578125, -0.26171875, -1.15625, -0.736328125, -1.6640625, -0.59912109375, -0.4365234375, -0.97802734375, 1.2607421875, 2.181640625, 0.6533203125, 0.351806640625, 0.283935546875, -0.21923828125, 0.6630859375, 0.9912109375, 1.8603515625, 0.07879638671875, -0.796875, 1.9345703125, 0.5888671875, -1.087890625, -0.0225067138671875, 1.3115234375, 0.366943359375, 0.318115234375, -0.5107421875, 0.923828125, -0.99169921875, 0.50390625, 1.5048828125, 0.1669921875, 0.31494140625, 0.376953125, 0.46435546875, 0.43212890625, 0.77978515625, 0.37109375, 0.736328125, 1.744140625, -0.63720703125, 0.9775390625, 0.11578369140625, 1.2802734375, 2.5234375, 0.0693359375, 1.5771484375, 0.04248046875, -0.243408203125, -1.1005859375, -0.7470703125, 0.099365234375, -1.40625, -0.08721923828125, 1.2099609375, 0.89501953125, 0.386962890625, -0.89111328125, -2.3125, -0.2276611328125, -0.07061767578125, 1.0283203125, -2.548828125, 1.05859375, -2.099609375, -0.63818359375, -0.99462890625, 0.91650390625, -1.857421875, 0.71435546875, 2.107421875, -0.352294921875, -1.4658203125, -0.06048583984375, -0.060455322265625, 0.70849609375, 0.1060791015625, -0.51123046875, 1.7314453125, -1.2197265625, 0.05010986328125, 0.353271484375, 0.263916015625, 0.1341552734375, 1.1181640625, -0.39111328125, -1.2265625, -0.71533203125, 1.0146484375, 0.150634765625, -1.0830078125, 0.222900390625, -1.1669921875, 0.250244140625, 0.230712890625, -1.76171875, 0.95556640625, 0.37158203125, -1.4755859375, 1.2275390625, 1.40234375, 1.05078125, -1.0458984375, 0.3212890625, -0.6494140625, -0.529296875, 0.434814453125, 0.39453125, 0.20947265625, 1.2958984375, -0.12164306640625, 0.73388671875, -3.00390625, -1.341796875, 1.44140625, -0.345458984375, -0.68310546875, 0.76806640625, -1.5078125, -1.0048828125, -1.263671875, 0.52978515625, -0.038787841796875, -1.0078125, 0.78466796875, -0.65673828125, 0.71240234375, 0.2086181640625, 0.89990234375, 0.341796875, 0.1087646484375, 0.1322021484375, -0.9453125, -2.865234375, 1.828125, 1.2978515625, -1.2314453125, -0.9091796875, 0.333740234375, 1.28125, 2.529296875, -2.572265625, 0.468994140625, -1.0244140625, 0.456298828125, -0.8916015625, 0.9072265625, 0.248779296875, 0.293212890625, -0.0819091796875, -1.865234375, 0.1805419921875, -0.049591064453125, -0.78271484375, -1.021484375, -2.07421875, 0.66357421875, 0.1817626953125, 1.1669921875, 1.23046875, -1.9892578125, -1.1962890625, 0.58349609375, 1.2158203125, -0.59716796875, -2.01171875, -0.77880859375, -0.59619140625, 0.2032470703125, -0.77783203125, 1.412109375, -0.1080322265625, 0.242431640625, -0.09320068359375, -0.5087890625, -0.242919921875, -0.4580078125, -0.83251953125, 0.73095703125, -0.51904296875, 1.625, 0.447509765625, 0.28125, 0.396728515625, -1.0615234375, -0.18115234375, -0.72216796875, 0.69775390625, 1.3212890625, 0.64990234375, -0.039459228515625, 1.072265625, -0.461181640625, -0.61279296875, -0.95263671875, -0.87060546875, -1.12890625, -1.4921875, 1.0078125, -0.9013671875, 1.048828125, 0.1275634765625, -1.1962890625, 0.619140625, 1.6318359375, 0.68408203125, 0.260986328125, -0.4091796875, 1.9375, 0.7509765625, -0.310302734375, -0.58447265625, -1.1162109375, 0.2044677734375, 1.7294921875, 0.88427734375, 0.69140625, -0.10760498046875, 0.061798095703125, 0.92578125, -0.55908203125, 0.3359375, -1.138671875, -0.099609375, 0.477294921875, -0.64111328125, -1.11328125, -0.375244140625, 0.092529296875, -0.74853515625, -1.806640625, 0.467529296875, 0.375244140625, -0.498291015625, 0.51708984375, 0.751953125, -0.50341796875, 0.4912109375, -0.8623046875, 0.1019287109375, -1.615234375, -0.09649658203125, -1.0791015625, -0.0211639404296875, -0.43310546875, 1.951171875, -0.23193359375, -1.5537109375, -1.1533203125, -0.6181640625, 0.32958984375, -1.2548828125, -2.326171875, 0.89111328125, -0.287353515625, 1.2587890625, 1.1201171875, 0.1895751953125, 0.2467041015625, 2.41015625, 0.646484375, 1.3701171875, -1.1552734375, 0.1925048828125, -0.0025730133056640625, 0.7607421875, 0.468994140625, 0.060791015625, -1.19140625, 0.320068359375, -1.3828125, -1.0947265625, -0.5849609375, -1.32421875, -0.00209808349609375, 0.697265625, 1.275390625, 1.0087890625, 1.7021484375, 1.9267578125, 1.2802734375, 0.80224609375, 0.15576171875, 0.1990966796875, -0.55078125, -2.32421875, -0.875, -1.142578125, 1.1796875, 1.3369140625, 0.7861328125, 0.195068359375, -0.3349609375, 0.5263671875, 0.1376953125, -0.421875, 0.7177734375, 0.2257080078125, 0.5419921875, 0.6435546875, 0.321533203125, -2.0703125, 1.7509765625, 0.1954345703125, -1.326171875, -0.313720703125, -0.99267578125, -0.587890625, -0.52099609375, 0.371337890625, -1.34765625, 0.8486328125, 2.11328125, -0.1458740234375, -1.4267578125, -0.537109375, 0.0280914306640625, 1.6875, 0.86669921875, 0.482666015625, 3.85546875, 1.4296875, -0.705078125, 0.2418212890625, 1.734375, 0.47802734375, 2.244140625, -0.450439453125, 0.53564453125, -1.3935546875, 0.73876953125, -0.2469482421875, 0.256103515625, -0.90283203125, 0.194091796875, 0.515625, 0.38134765625, -1.0263671875, 1.0498046875, -0.438232421875, 1.2451171875, -0.67333984375, 1.98046875, 0.9267578125, -2.1328125, 0.350341796875, -0.2489013671875, -1.73828125, -0.37158203125, -0.6123046875, -2.90234375, -0.8916015625, -0.26806640625, -1.453125, -2.51953125, 0.8173828125, 0.471435546875, 0.8173828125, -0.9580078125, -1.3564453125, 0.06781005859375, -0.1265869140625, -0.5703125, 0.99462890625, -1.333984375, 1.1220703125, 1.3603515625, 0.658203125, 0.0338134765625, 0.73388671875, -1.8916015625, -0.0482177734375, -0.32080078125, -0.341796875, -0.205078125, 0.74951171875, 0.230224609375, -0.2939453125, -1.1484375, -0.68505859375, 0.37451171875, 0.775390625, 0.46728515625, 1.2890625, 1.1533203125, -1.7412109375, -0.6318359375, 0.435302734375, -0.054779052734375, 1.1044921875, -0.06732177734375, 0.56982421875, 0.466064453125, 0.141357421875, -0.08953857421875, 1.2919921875, -1.5068359375, -1.455078125, 1.1435546875, 1.4208984375, -0.96044921875, -1.59375, 0.0150146484375, -0.76123046875, 0.59619140625, 1.02734375, 1.8076171875, 0.272705078125, 0.515625, 0.45166015625, 0.260009765625, 0.623046875, -1.0654296875, 0.92529296875, -0.179443359375, 1.0205078125, -0.12939453125, 0.493896484375, 0.43896484375, 0.71533203125, 0.55419921875, 0.32177734375, 1.203125, 1.279296875, 0.419921875, -0.958984375, -1.2861328125, 0.464111328125, 1.2109375, -3.072265625, -1.470703125, 0.77099609375, 0.443359375, -0.0155181884765625, 1.431640625, -0.488037109375, -0.6943359375, 0.73291015625, -0.1768798828125, 0.287841796875, 0.22998046875, 0.86669921875, 0.98583984375, -0.80322265625, 1.3134765625, 0.58349609375, 1.2216796875, -1.640625, 1.048828125, 0.69580078125, 1.0537109375, -0.568359375, 0.619140625, -0.66845703125, 0.54150390625, 0.2467041015625, 0.7236328125, -0.30517578125, -0.84619140625, 0.33447265625, 0.97509765625, -0.56689453125, -0.0655517578125, 0.81787109375, -0.2210693359375, 1.1337890625, -0.38671875, -0.453857421875, 0.047454833984375, 0.95703125, 0.8017578125, -1.0966796875, 2.0859375, -1.2666015625, -1.2021484375, 0.43359375, 1.384765625, 0.04046630859375, 0.0013341903686523438, 1.0888671875, -1.255859375, 0.76025390625, -0.86669921875, 0.13330078125, -0.2264404296875, -1.193359375, -2.314453125, -0.3427734375, 0.64794921875, 0.5576171875, 0.32421875, -1.2734375, 0.351806640625, -1.2060546875, 0.7490234375, 0.49365234375, -1.7587890625, 0.0252685546875, 0.80078125, 0.85498046875, 0.94091796875, -1.6396484375, 0.142822265625, 0.888671875, -1.9072265625, -1.259765625, -0.0343017578125, -2.576171875, -0.931640625, -1.6982421875, -1.0458984375, 0.275390625, -0.61376953125, -0.83984375, 1.013671875, 0.3212890625, -0.096435546875, 0.69873046875, -1.091796875, 0.8701171875, 2.30078125, -0.011505126953125, 0.0311431884765625, -0.09661865234375, 0.248291015625, 1.3798828125, -0.88134765625, -0.7353515625, 0.1358642578125, 1.61328125, -0.5478515625, -1.150390625, 0.6669921875, -0.11700439453125, 0.5771484375, 1.15625, -0.67529296875, 0.92041015625, 0.9296875, -1.265625, 0.61865234375, 0.111083984375, 0.08941650390625, -1.0927734375, -0.3359375, -0.08251953125, -0.12939453125, 1.40234375, 0.9716796875, 0.412841796875, -0.93115234375, 0.1077880859375, -1.791015625, -0.37890625, -0.69775390625, 1.0400390625, -0.72900390625, 1.353515625, 1.7822265625, 0.024871826171875, -1.533203125, 0.9423828125, 0.70947265625, -0.392578125, 1.2861328125, -0.133544921875, -0.642578125, -0.2073974609375, 0.330078125, -1.8623046875, 0.263427734375, 2.41015625, -0.8671875, -0.423095703125, 0.389892578125, -0.06463623046875, -1.15234375, -0.66259765625, -2.46484375, -1.5771484375, -0.62646484375, -0.9931640625, -0.59521484375, -1.060546875, -0.10369873046875, -0.2578125, 0.329833984375, 1.115234375, 2.01171875, -0.07421875, -0.98291015625, -1.494140625, -0.95703125, 0.8779296875, 0.73388671875, -0.160888671875, 0.5029296875, -0.509765625, 0.6455078125, 1.9208984375, 1.287109375, -1.08984375, 0.5029296875, -2.4609375, 0.0010519027709960938, 0.094970703125, -0.89892578125, -0.50634765625, -0.1668701171875, -2.1875, 0.399658203125, 0.10101318359375, 0.564453125, -1.357421875, -0.26318359375, 0.45703125, -0.72705078125, -1.041015625, -0.08514404296875, 0.9150390625, -0.337890625, -0.900390625, -0.378173828125, 0.716796875, -2.84375, -0.25244140625, 0.6455078125, 0.58984375, 0.7529296875, 1.4033203125, -0.82568359375, 1.044921875, 1.10546875, 1.3203125, -1.0009765625, 0.6513671875, 0.00475311279296875, -0.8564453125, 0.6796875, 0.46923828125, -0.89599609375, 0.3603515625, 1.1611328125, -0.374267578125, -0.98388671875, -0.2252197265625, 1.38671875, 0.62744140625, -0.8525390625, 0.7880859375, 1.4765625, 2.005859375, -0.14794921875, 0.65673828125, -0.1107177734375, 1.193359375, 0.41357421875, 0.4814453125, -1.47265625, -1.41796875, -0.1546630859375, 1.001953125, -0.51611328125, 0.794921875, 2.220703125, 1.3828125, -1.27734375, 0.08056640625, -0.197998046875, -0.634765625, -0.0975341796875, -0.141845703125, -3.119140625, 0.6953125, -0.1737060546875, -1.5517578125, -0.332763671875, 1.6279296875, -0.437744140625, 0.2293701171875, -1.689453125, 0.437744140625, 1.310546875, -1.0048828125, 0.10821533203125, -0.58984375, 0.27099609375, 1.994140625, 0.42626953125, -1.4365234375, -0.7734375, -0.228515625, 2.009765625, 0.46533203125, -1.025390625, 0.56298828125, 1.4345703125, 0.189208984375, -0.74609375, -0.76416015625, 1.2587890625, -0.37841796875, 2.0546875, -0.367431640625, -0.64697265625, 1.017578125, 0.03076171875, 1.0849609375, -1.3193359375, 0.42626953125, -1.0537109375, -0.90478515625, 0.1385498046875, -0.35791015625, -0.0206146240234375, 1.4306640625, 0.313720703125, -0.257080078125, -0.01444244384765625, -1.5517578125, -1.0341796875, -0.67333984375, 0.80615234375, 1.4580078125, -0.88427734375, -1.55078125, -0.387939453125, -0.364501953125, 0.1610107421875, -1.138671875, -1.041015625, 0.1312255859375, 1.33984375, -1.2607421875, -1.349609375, -0.34326171875, -0.62255859375, 0.61083984375, 1.3984375, 0.207275390625, -0.92138671875, -0.89453125, 1.0107421875, -0.51220703125, 0.272705078125, -0.9208984375, -0.451904296875, -0.5966796875, -0.0241546630859375, -1.216796875, -0.94873046875, -0.7255859375, -1.6513671875, 0.25390625, -2.232421875, -1.3828125, -1.2705078125, -0.4150390625, -1.5791015625, -2.236328125, 0.89111328125, -2.765625, -1.015625, -0.34423828125, 0.295654296875, 0.08233642578125, 0.5595703125, 2.400390625, -0.08892822265625, -0.9853515625, 0.19384765625, -0.595703125, -0.050689697265625, -0.9091796875, 0.541015625, 0.032562255859375, -1.0771484375, 0.52734375, -2.640625, -1.7587890625, 0.0865478515625, 1.05078125, -1.4443359375, 0.873046875, -0.51806640625, -0.364013671875, -0.7412109375, -1.2451171875, -0.61669921875, -1.66015625, -0.53515625, 0.2449951171875, -0.40966796875, -1.451171875, 0.2191162109375, -0.039398193359375, -1.625, -1.021484375, 0.57958984375, -0.7041015625, -0.3212890625, -2.744140625, 1.943359375, -0.7333984375, 1.94140625, -0.64208984375, 0.83203125, 1.7373046875, 0.1318359375, 0.30615234375, 0.1376953125, -1.0986328125, -1.720703125, 1.2626953125, -0.2239990234375, -1.91015625, 0.64453125, -0.2362060546875, -0.30517578125, -0.37939453125, -1.669921875, -0.3291015625, 1.0029296875, -0.68896484375, -0.415771484375, 0.11773681640625, 0.77294921875, 1.107421875, 1.1015625, -1.1572265625, -0.1451416015625, 0.837890625, 0.295166015625, -1.3642578125, 1.279296875, -1.2802734375, 0.62109375, -0.72314453125, 0.638671875, -0.346923828125, -1.47265625, -0.48193359375, 1.353515625, 0.305419921875, -1.7900390625, 0.04840087890625, 0.1654052734375, 0.1650390625, -0.2452392578125, -1.8935546875, 0.2149658203125, -0.306884765625, -0.397705078125, -0.1298828125, -1.599609375, 1.5068359375, -0.316650390625, 0.431884765625, -1.560546875, -1.0888671875, -0.78173828125, -0.36083984375, -0.89794921875, -0.75732421875, 0.71484375, 3.294921875, 1.482421875, -1.3037109375, -1.4990234375, 0.7802734375, -0.86279296875, -0.97119140625, 0.1883544921875, 0.6953125, 1.326171875, -0.7236328125, -0.0310516357421875, -0.474365234375, 1.0390625, 0.1373291015625, -1.8447265625, 1.025390625, -2.478515625, -0.9228515625, 0.841796875, 1.986328125, -1.3486328125, -0.87353515625, -0.69189453125, -0.1572265625, -0.56787109375, 0.75390625, 0.8154296875, -0.495361328125, -0.271240234375, 0.0733642578125, 0.65087890625, 0.57373046875, 0.6796875, -1.33984375, -1.666015625, -0.87109375, -0.01374053955078125, 0.9052734375, 2.013671875, -1.263671875, -1.0625, 0.1912841796875, -1.8701171875, -0.8701171875, -1.35546875, 0.367919921875, 0.007694244384765625, 1.1611328125, -0.927734375, 1.615234375, 1.0908203125, -1.4970703125, 1.083984375, -0.9521484375, -0.42333984375, 2.419921875, -0.91357421875, 1.0966796875, 0.285888671875, 0.66015625, -0.80859375, -1.849609375, 0.08648681640625, 1.41796875, 0.37646484375, 0.6396484375, 0.029144287109375, -0.79443359375, -0.7353515625, 1.537109375, 0.1910400390625, 1.078125, -1.11328125, 0.1832275390625, -0.52294921875, 1.546875, -0.416259765625, -0.041961669921875, 0.2064208984375, -0.7685546875, -0.68701171875, 0.546875, 0.39501953125, 0.221923828125, -1.91796875, -0.76904296875, 1.318359375, -1.52734375, 1.2099609375, -1.158203125, 2.111328125, -0.83203125, 0.87060546875, 0.70361328125, 0.080078125, 0.52001953125, 1.5341796875, -0.6787109375, -0.95751953125, -0.85986328125, 0.448486328125, 0.290283203125, 0.77587890625, -1.0947265625, 1.5390625, 1.400390625, -1.49609375, -0.71240234375, 0.490234375, -0.0211944580078125, -0.37255859375, 0.26953125, -0.6513671875, -1.69140625, 1.1005859375, 0.3466796875, 0.29931640625, 1.8193359375, 0.65283203125, 0.01338958740234375, 1.4140625, 3.060546875, -0.12353515625, 0.52197265625, 0.73291015625, 2.25, -0.2408447265625, 1.0927734375, 1.755859375, 2.28125, 0.7783203125, -1.17578125, 0.59228515625, 1.607421875, 1.0205078125, -0.42529296875, -0.763671875, -0.85498046875, 0.480224609375, -1.93359375, 0.703125, 0.105224609375, -1.7880859375, -0.2724609375, 0.1866455078125, -0.85546875, -1.591796875, 0.86083984375, -0.94091796875, 1.453125, -1.3193359375, -0.9765625, -0.1451416015625, -0.269287109375, -0.595703125, -0.439208984375, 2.34765625, 0.97998046875, -0.2222900390625, 0.2125244140625, 0.373291015625, 0.7568359375, -1.2724609375, 0.96728515625, 0.69287109375, -0.213134765625, -0.426025390625, 1.14453125, -0.76318359375, -1.1806640625, -0.0152130126953125, -1.1162109375, 1.0068359375, 0.79931640625, -0.81591796875, -0.93701171875, 0.88525390625, 0.60693359375, -0.10943603515625, -0.48681640625, 2.10546875, 0.03668212890625, 0.76611328125, -0.52734375, -1.451171875, 0.2022705078125, -1.095703125, 1.0107421875, 1.03125, -0.2052001953125, -2.083984375, 0.09814453125, 1.4462890625, 0.82177734375, 1.921875, -0.94482421875, 1.46875, -0.853515625, 1.21875, 2.01171875, -0.44921875, -0.9375, 0.61572265625, 0.2490234375, 1.705078125, -0.5830078125, -0.318359375, -0.51318359375, 0.78076171875, 1.642578125, -0.71337890625, 0.29443359375, 1.482421875, -0.58984375, 0.1475830078125, 0.28564453125, 1.1123046875, -0.8232421875, -0.43310546875, -2.06640625, 0.69287109375, 2.40234375, -0.052825927734375, -2.11328125, -0.77685546875, 0.352294921875, 0.07867431640625, 0.8837890625, -0.392822265625, 0.4794921875, -0.458984375, 1.0380859375, -1.30859375, 0.52734375, -0.203125, 0.56982421875, -1.314453125, 1.27734375, -0.46923828125, 1.099609375, 0.7578125, 1.166015625, 0.094482421875, -0.9052734375, 1.2490234375, 2.12109375, -0.351318359375, -0.744140625, -1.3115234375, 0.378173828125, 0.74267578125, 0.91552734375, -0.771484375, 0.44921875, -0.12164306640625, -0.09417724609375, 0.0479736328125, -0.69970703125, -1.0498046875, -2.080078125, -0.350341796875, 1.8427734375, -1.04296875, -0.943359375, -0.9501953125, -1.6728515625, 0.6083984375, -0.09356689453125, 0.72265625, 0.112060546875, -0.16259765625, -0.1265869140625, -1.7724609375, -0.1578369140625, 1.7734375, -0.80078125, 0.92138671875, -0.36279296875, 0.546875, 1.1875, -1.033203125, 1.45703125, 0.242431640625, -0.95263671875, -0.2471923828125, 0.9990234375, 0.4287109375, 0.208984375, 0.27783203125, 1.4501953125, -0.453369140625, -0.2286376953125, -1.8583984375, 0.02093505859375, -0.0498046875, 0.29638671875, 0.34619140625, 1.740234375, -1.61328125, -0.022003173828125, 0.06256103515625, -0.59814453125, -1.947265625, -0.356201171875, 1.03515625, 0.71630859375, -0.83740234375, 1.328125, 2.376953125, -0.74072265625, 0.966796875, 0.39794921875, 0.324462890625, -0.37255859375, 0.04364013671875, 1.2568359375, 0.184326171875, -0.166015625, 0.453125, -0.458740234375, -0.912109375, -1.31640625, 0.20556640625, -0.49560546875, 1.626953125, 0.65087890625, -1.2490234375, -0.29638671875, 2.060546875, 2.90625, 1.8603515625, 0.53857421875, -0.82177734375, 0.0276641845703125, 0.0987548828125, -1.3173828125, -0.2261962890625, -0.99658203125, -0.37109375, 1.677734375, -2.1796875, -2.544921875, 0.08587646484375, -1.634765625, 1.38671875, -0.56640625, -0.400390625, -1.576171875, 1.404296875, 1.3076171875, 0.8994140625, 0.302001953125, 0.53515625, -0.277099609375, -1.203125, 0.1715087890625, 0.027679443359375, 0.51416015625, -0.1343994140625, -0.1226806640625, -1.974609375, -1.5146484375, -0.271240234375, 0.810546875, -0.54443359375, -1.1259765625, -1.49609375, -1.7578125, 0.2529296875, -0.396728515625, -0.1522216796875, 0.3408203125, -0.5771484375, -0.468017578125, 1.6875, 0.1463623046875, 1.2392578125, -0.4169921875, -0.27490234375, -0.387939453125, 0.32568359375, -0.3056640625, -0.411376953125, -0.54443359375, 0.52685546875, -1.017578125, 0.9208984375, 1.552734375, 0.26513671875, 0.039886474609375, -0.43994140625, 0.703125, 1.0634765625, 2.064453125, 0.63232421875, 0.040863037109375, -0.2349853515625, 0.1063232421875, -0.2242431640625, -0.755859375, 0.50341796875, -0.267822265625, -1.0888671875, -0.67626953125, -0.257080078125, 1.080078125, 0.73095703125, -1.197265625, 0.420654296875, 0.6220703125, 0.73681640625, 0.63037109375, 0.328857421875, 1.6123046875, -0.98486328125, 0.04168701171875, 0.6455078125, -1.1513671875, -0.4541015625, 0.5966796875, -0.1781005859375, -2.14453125, 1.265625, 0.265380859375, 0.55029296875, -0.047332763671875, 1.8681640625, 0.10888671875, 0.38671875, -1.056640625, -1.6142578125, 1.5966796875, -0.373291015625, -0.045135498046875, -2.66796875, 0.435302734375, 0.295654296875, -0.8876953125, 0.53369140625, 0.9033203125, -0.9169921875, 0.73095703125, 0.2978515625, 0.51611328125, 0.384765625, 0.78564453125, 0.1885986328125, 1.796875, 0.435546875, -2.3359375, -0.032806396484375, 0.74072265625, 0.1812744140625, -0.2197265625, -1.6416015625, -2.0234375, 0.04638671875, 1.7841796875, -0.61865234375, 1.8408203125, 0.79833984375, -2.046875, -0.0202484130859375, 0.2442626953125, -2.466796875, 0.151123046875, -0.36474609375, 0.69140625, -0.46728515625, 1.0341796875, 0.67724609375, -0.1490478515625, -0.458251953125, 1.4619140625, 1.2294921875, -0.62841796875, 0.0928955078125, -0.056060791015625, -1.8515625, 0.55810546875, 0.9912109375, -1.94140625, 1.0859375, -0.86962890625, -0.27978515625, 0.50634765625, 0.53662109375, 1.2783203125, 1.869140625, -1.2255859375, -1.3515625, 0.94921875, -1.193359375, 0.6396484375, 0.58984375, -1.3056640625, -2.123046875, 0.84130859375, -1.3642578125, 0.293701171875, 1.421875, -0.2261962890625, -0.3125, 0.7587890625, -0.1429443359375, 0.63134765625, 0.181640625, 1.7177734375, -0.8427734375, -0.662109375, -0.50244140625, -0.10675048828125, -1.1171875, -0.292724609375, 0.318603515625, -0.025360107421875, -0.1929931640625, -0.6806640625, 0.87841796875, -1.3203125, 0.265869140625, -1.263671875, -0.5400390625, 1.2001953125, 0.958984375, -0.082275390625, 0.1475830078125, -0.658203125, 1.259765625, -0.2490234375, -0.56396484375, 0.25830078125, -0.859375, -0.114013671875, 0.422119140625, 2.0, 0.68359375, -0.60595703125, 0.67919921875, -2.44921875, -0.35400390625, 1.908203125, 1.0859375, 1.517578125, -0.10858154296875, 0.248779296875, -0.8935546875, 0.382080078125, 0.289794921875, -1.9228515625, 1.4736328125, 1.4638671875, -0.489990234375, -0.994140625, -0.344482421875, 0.66845703125, 0.492919921875, 1.7421875, 1.9375, -3.048828125, -0.70263671875, 0.83642578125, -0.01947021484375, -0.5732421875, 0.1817626953125, -1.109375, -0.94921875, 0.56494140625, -0.6865234375, -0.89501953125, 1.025390625, -0.1697998046875, -0.11322021484375, -0.499755859375, -1.513671875, -0.226318359375, -0.289794921875, 0.77490234375, -0.424072265625, 1.1953125, 0.6845703125, 0.289794921875, 1.169921875, -0.08587646484375, 0.8349609375, 1.697265625, 0.339111328125, 0.129150390625, -0.498291015625, -0.23876953125, -1.7099609375, -0.68017578125, 1.603515625, -0.284423828125, 0.37744140625, 0.7587890625, -0.300048828125, -1.703125, -1.08984375, -0.179443359375, -2.166015625, -0.48828125, -0.07489013671875, 0.146728515625, 1.0615234375, -0.245849609375, 0.73193359375, -0.72705078125, -0.5166015625, 0.463134765625, -0.724609375, -0.0633544921875, 0.8828125, 0.384521484375, -0.6611328125, -0.6796875, 1.4794921875, 0.7666015625, -0.499267578125, -0.4638671875, 1.369140625, -1.5283203125, 0.33837890625, 0.07562255859375, -0.47021484375, -0.50927734375, 1.998046875, 0.95361328125, 0.4296875, 1.2783203125, -1.6337890625, -1.091796875, 1.18359375, 1.2392578125, 0.390380859375, -0.7265625, 0.489990234375, 0.263671875, -0.53515625, -1.1982421875, 0.379638671875, 1.2939453125, -2.83203125, 1.54296875, 0.7607421875, -1.103515625, -0.2012939453125, -1.083984375, -1.6552734375, 0.69677734375, -0.1083984375, -0.156005859375, -0.6826171875, -0.54296875, 1.5302734375, 0.2315673828125, 1.1044921875, -0.62841796875, -1.5283203125, -0.5556640625, -1.2734375, -0.27783203125, -1.5947265625, 0.0823974609375, -0.2174072265625, -0.482177734375, -0.1268310546875, 0.0570068359375, 0.57958984375, -0.52734375, 0.60205078125, -0.71875, 0.0028095245361328125, -1.38671875, -2.0859375, -1.388671875, 1.109375, 0.1885986328125, -0.298828125, 0.4375, 0.346923828125, 0.025634765625, -0.0684814453125, 0.477294921875, 0.65966796875, -0.449462890625, 0.333984375, 0.493896484375, -0.316162109375, 1.447265625, -0.14208984375, 0.828125, -0.1336669921875, 2.484375, -0.8974609375, -0.1400146484375, 2.623046875, 1.0869140625, 0.0340576171875, -0.1531982421875, 0.58447265625, -0.1641845703125, 0.1611328125, 0.96044921875, 0.485107421875, -0.331298828125, 1.625, 0.76904296875, 0.5751953125, -0.4404296875, -1.451171875, -0.033538818359375, 1.0908203125, -0.2137451171875, -0.97998046875, 0.433837890625, 0.79638671875, -0.1583251953125, -0.3583984375, -0.3671875, 0.74169921875, -0.91650390625, 0.72802734375, -0.2236328125, -0.6865234375, 0.0205230712890625, -0.285888671875, 0.15478515625, 0.87744140625, -1.9013671875, 0.6904296875, 0.064208984375, -0.02801513671875, -0.96484375, 0.1282958984375, -0.796875, 2.162109375, 0.437255859375, 1.2177734375, 0.0987548828125, -0.453369140625, 1.90625, 0.7236328125, 0.058258056640625, -0.0511474609375, 0.84326171875, 0.73876953125, -0.368896484375, -0.62109375, -0.1287841796875, 0.4248046875, -1.1962890625, 0.194580078125, 0.1319580078125, 0.495361328125, 2.10546875, 0.841796875, 0.0882568359375, 0.2257080078125, -0.9736328125, -0.59716796875, 0.91259765625, 0.432373046875, 0.278564453125, -0.14208984375, 1.21875, -0.75732421875, -0.3671875, 0.775390625, 0.05322265625, -0.34326171875, -0.32421875, -0.64892578125, 0.462646484375, 0.1402587890625, 0.399658203125, 0.296630859375, 1.2001953125, 1.51953125, -0.53564453125, -0.491943359375, -1.0712890625, 1.2060546875, 0.94287109375, -0.5126953125, -0.63427734375, -0.309814453125, -0.9736328125, -0.304931640625, 1.0234375, -1.4853515625, -0.2027587890625, -0.5224609375, 0.2352294921875, -0.1318359375, -0.2100830078125, -1.1376953125, 0.1534423828125, 1.1806640625, -1.701171875, 0.71337890625, 1.2490234375, -0.583984375, -0.302001953125, 2.166015625, 0.64404296875, 1.05078125, 1.580078125, -0.284423828125, -1.05859375, 0.6748046875, -1.087890625, 1.076171875, 0.91357421875, -1.2333984375, 1.0078125, 0.91796875, -0.4384765625, 0.9609375, 0.5283203125, -1.427734375, 0.72412109375, -1.478515625, -0.160888671875, 1.3115234375, 0.142333984375, 1.328125, -1.380859375, 0.68212890625, 0.170654296875, -1.9423828125, -0.8876953125, -0.8896484375, -0.54541015625, 0.05560302734375, -0.061004638671875, 1.2509765625, 0.75537109375, -2.390625, -1.5693359375, 1.0654296875, -1.3515625, -1.8173828125, 0.283935546875, 0.67626953125, -0.9521484375, -1.546875, 0.1351318359375, 1.0107421875, 0.00274658203125, -1.0361328125, -0.408935546875, -0.63916015625, -0.72412109375, 0.98681640625, 1.3984375, -0.07275390625, 0.257080078125, 0.57373046875, 0.268310546875, 0.446044921875, 0.7119140625, -0.0148773193359375, -0.177490234375, 1.07421875, 0.1634521484375, 2.126953125, -1.4482421875, -0.05206298828125, 0.70166015625, -1.869140625, -0.76025390625, -1.6220703125, -1.8525390625, -1.1328125, 1.357421875, 0.1641845703125, -0.37158203125, 0.93896484375, -0.1954345703125, -1.0517578125, -0.6533203125, -0.5390625, 1.01171875, -1.8193359375, 0.0511474609375, 0.07763671875, -1.8935546875, -0.79833984375, -0.489501953125, -0.62646484375, 0.9619140625, -0.2353515625, -0.1324462890625, 0.33740234375, -0.494140625, 0.1553955078125, -0.35693359375, -0.951171875, -1.29296875, -0.2249755859375, -1.818359375, -1.6435546875, -1.0361328125, 0.53515625, 0.351318359375, -0.324951171875, -1.01171875, 0.9228515625, -1.2880859375, -0.380859375, -1.43359375, -0.56982421875, 0.01003265380859375, 1.275390625, -0.8232421875, 0.9892578125, 0.57421875, -0.54638671875, 1.21484375, 0.062744140625, 0.1802978515625, 0.84375, 0.83935546875, 0.40625, 1.7841796875, -1.6904296875, -1.1494140625, 0.10455322265625, -0.99755859375, -0.267578125, -0.9931640625, -0.174072265625, 1.83203125, 0.39697265625, -0.0770263671875, -1.03125, -0.92529296875, 0.71923828125, -0.312744140625, 1.3330078125, -1.58203125, 0.69873046875, 0.468994140625, -0.412353515625, 0.045135498046875, -0.029327392578125, -1.1015625, -0.2091064453125, -0.58642578125, 0.369384765625, -0.87744140625, -0.64306640625, -0.0012760162353515625, 0.671875, 1.5244140625, -0.468994140625, 1.5888671875, -1.041015625, -1.8115234375, -0.86279296875, 1.4609375, 0.266845703125, -0.578125, -0.498291015625, -0.341552734375, -1.4287109375, 0.52880859375, -1.0029296875, 1.4169921875, 1.4580078125, -0.44970703125, 0.4580078125, 0.494384765625, -0.28662109375, -1.6044921875, 0.97412109375, -1.0283203125, -0.1353759765625, -0.68115234375, 1.193359375, 0.603515625, -0.384033203125, -0.27294921875, 0.349853515625, 0.67919921875, 1.6865234375, -0.262939453125, -1.42578125, -1.2158203125, 0.904296875, 0.736328125, -0.63330078125, 0.64453125, 1.3876953125, 1.5537109375, -0.16357421875, -1.0869140625, -0.224853515625, 0.1959228515625, -1.3388671875, 0.73486328125, 0.2200927734375, 0.388427734375, 0.033538818359375, -0.47265625, -0.47412109375, 0.619140625, -0.06549072265625, 0.65234375, -0.66357421875, 0.8623046875, -0.28857421875, 0.37060546875, 0.375732421875, -0.8212890625, 1.26953125, -0.529296875, -1.009765625, -0.40283203125, -0.98779296875, 1.8828125, 0.439697265625, -0.79296875, 0.35009765625, 0.626953125, 0.2548828125, -1.1669921875, -0.921875, -0.68115234375, -1.0439453125, -1.2412109375, 0.5380859375, -0.9443359375, -0.88623046875, 0.038848876953125, -1.365234375, 0.264892578125, -0.09765625, 0.0428466796875, 1.4736328125, -0.4658203125, -1.2265625, 0.56103515625, 1.001953125, 3.361328125, 2.470703125, -0.94189453125, 0.95751953125, -1.2197265625, 1.7041015625, -0.3916015625, 1.287109375, 0.3720703125, 0.2423095703125, 1.353515625, -0.51806640625, -1.140625, -0.1934814453125, -1.2919921875, -0.42529296875, -0.362548828125, 0.56103515625, 0.24169921875, -1.80078125, -0.27490234375, -0.51953125, -1.0537109375, -0.0997314453125, -0.068359375, -0.8740234375, -0.260009765625, -1.2900390625, 0.478271484375, -0.75927734375, -0.068603515625, 0.70947265625, 2.013671875, 0.78125, -0.1572265625, -0.259521484375, 0.916015625, 0.6044921875, 0.278076171875, 0.370849609375, 2.39453125, 0.87353515625, -0.303466796875, -0.1678466796875, -0.50732421875, -0.55126953125, 2.08203125, -0.9482421875, -0.62158203125, 0.273193359375, -0.2030029296875, 0.77685546875, 0.62939453125, 0.9091796875, -0.52294921875, 0.444580078125, -0.6689453125, -0.1866455078125, -1.2509765625, -1.2978515625, -0.39404296875, 1.5302734375, 0.580078125, -0.445556640625, -1.4541015625, -0.199951171875, -0.63427734375, 1.431640625, -1.201171875, -0.08819580078125, -1.4306640625, -1.55859375, -0.7919921875, 1.0498046875, 1.7841796875, -0.947265625, -0.14990234375, -1.2626953125, -0.302734375, -0.157958984375, -1.55078125, -1.283203125, -0.79833984375, 2.41015625, 0.046844482421875, 2.103515625, 2.22265625, 2.265625, -0.128662109375, -0.623046875, -0.254150390625, -0.448974609375, 1.76171875, -1.0341796875, 0.185791015625, 0.8330078125, -0.900390625, 1.595703125, 0.375, 0.405517578125, 0.186279296875, 1.44921875, -1.37890625, -1.0966796875, -1.4609375, -1.3408203125, 0.468994140625, -2.6640625, -1.2373046875, -0.55908203125, 1.296875, -0.406494140625, -0.4482421875, -1.51953125, 0.2220458984375, -1.294921875, -1.267578125, 1.1953125, 0.876953125, -0.1024169921875, 1.3349609375, 0.77001953125, 0.568359375, 0.92919921875, -1.2607421875, -0.1875, -0.90576171875, 1.7119140625, -1.5869140625, 1.0380859375, 0.005977630615234375, 0.3916015625, 1.1669921875, 0.70654296875, 0.316162109375, -1.181640625, -0.352783203125, 0.9189453125, -1.0361328125, 0.6044921875, -0.254638671875, 0.048828125, 1.3984375, 0.9921875, 0.88671875, 0.04168701171875, -0.2626953125, -0.7001953125, -1.6201171875, -1.294921875, -0.5361328125, 0.9716796875, 0.62255859375, 1.330078125, 1.0986328125, -2.146484375, 1.5126953125, 0.97607421875, -0.779296875, 0.720703125, -1.6923828125, 0.498779296875, -0.81005859375, 0.037506103515625, 0.56103515625, -0.5419921875, 0.74609375, 1.3056640625, 0.453857421875, -1.74609375, -0.031402587890625, 1.7041015625, 0.07720947265625, 0.05645751953125, 2.693359375, -0.2244873046875, 0.433837890625, -0.306884765625, 0.58935546875, 1.12109375, -0.351318359375, 0.035552978515625, 0.7265625, 0.10845947265625, 0.9501953125, -1.6220703125, -0.6748046875, 0.388916015625, -0.056976318359375, -0.61572265625, -0.8291015625, 0.34130859375, 0.52880859375, 1.560546875, -1.228515625, 2.181640625, 0.484619140625, 0.85791015625, 0.1904296875, 0.330322265625, 1.154296875, 0.5810546875, 2.265625, 1.4208984375, -1.517578125, 1.1259765625, -0.473876953125, -0.2139892578125, -0.58544921875, -0.37939453125, 1.2578125, 0.020263671875, -0.45068359375, 0.39697265625, 1.2705078125, -1.173828125, -1.1806640625, 0.5615234375, 0.7890625, 2.009765625, -1.4892578125, 0.80712890625, -0.50537109375, -2.705078125, 2.349609375, -0.4423828125, -0.97119140625, -1.2646484375, 0.477783203125, 1.7392578125, 0.1434326171875, -0.243896484375, -0.33056640625, -0.78759765625, -1.4189453125, -0.72119140625, -0.1993408203125, -1.515625, 1.265625, 2.03125, -0.42724609375, 0.322265625, -0.39501953125, 0.79638671875, 0.4267578125, 0.66943359375, 0.46923828125, -1.779296875, 0.0081939697265625, 0.54931640625, -0.8193359375, -1.568359375, -0.76025390625, 0.65234375, -0.86279296875, -0.287109375, -1.0908203125, 0.338623046875, -0.26171875, 0.845703125, 0.7861328125, -0.6337890625, 0.63720703125, -1.28515625, 0.671875, -1.63671875, -0.22705078125, -0.21044921875, 0.64013671875, -0.88427734375, -0.245361328125, 0.380859375, -0.859375, -1.1875, 0.09332275390625, 1.0478515625, 0.1583251953125, -1.115234375, 0.533203125, 2.177734375, -0.41259765625, 0.499267578125, -0.55078125, -1.4541015625, -0.1595458984375, 0.765625, 1.4755859375, 0.46630859375, -0.224365234375, 1.3330078125, 0.1297607421875, -0.7822265625, 0.953125, 0.80712890625, 0.45068359375, 0.33154296875, 0.1553955078125, -0.479736328125, 1.279296875, 0.7392578125, -0.64599609375, -0.295166015625, 0.32275390625, -0.383544921875, 0.96826171875, 0.64599609375, 0.642578125, 0.299560546875, -1.8564453125, 2.14453125, -0.5341796875, -2.44140625, 0.4052734375, -0.52197265625, -0.80224609375, -1.193359375, 0.96484375, -1.103515625, -0.1722412109375, 0.87255859375, 0.7412109375, 1.3037109375, -0.921875, 0.912109375, 0.263916015625, -0.60498046875, -0.272705078125, 2.078125, -1.4208984375, -0.46337890625, 0.0135498046875, -0.990234375, 0.5087890625, 1.609375, -0.09649658203125, -0.7333984375, -0.68896484375, -1.0712890625, 2.001953125, -0.97607421875, -0.368408203125, -0.2685546875, -0.724609375, -1.37890625, -1.5234375, 0.8330078125, 0.703125, -0.66943359375, 0.495361328125, -0.2308349609375, -0.08831787109375, 0.564453125, -0.11962890625, -0.07452392578125, -0.4140625, 0.60498046875, 0.1279296875, 1.01953125, 0.143798828125, -0.634765625, 0.043365478515625, 0.434814453125, 0.705078125, -0.432373046875, -1.0126953125, -0.23974609375, -0.60009765625, 0.7734375, 1.0439453125, -0.7734375, 1.8173828125, 2.0078125, -0.272705078125, 0.61767578125, 2.076171875, 0.0699462890625, -0.08306884765625, -0.10235595703125, 0.91064453125, 1.00390625, 1.7861328125, -1.671875, -0.125732421875, -0.58154296875, 1.6162109375, 0.1593017578125, -1.6181640625, -1.201171875, 0.24072265625, -1.64453125, 0.331298828125, -0.064453125, -0.8994140625, 0.2408447265625, -1.3056640625, -0.36181640625, -0.736328125, 0.5419921875, 1.0498046875, -1.7958984375, 1.427734375, -0.3203125, 0.31884765625, -0.50341796875, -0.100830078125, 1.7333984375, 2.04296875, -1.138671875, 1.162109375, 1.1337890625, 0.1925048828125, -1.892578125, 1.6484375, -1.5283203125, 0.436767578125, 0.408447265625, 0.4384765625, -0.14453125, -1.173828125, 1.0439453125, 0.93505859375, 1.025390625, 1.4951171875, 0.95556640625, 0.1279296875, 0.84521484375, -0.830078125, -0.84521484375, -0.282958984375, -0.8740234375, -0.64697265625, -1.3017578125, 0.1087646484375, -1.150390625, 1.0771484375, -0.826171875, 2.154296875, 0.0550537109375, -0.428466796875, 1.2197265625, 0.352294921875, -0.08892822265625, -0.54736328125, -1.0654296875, 0.028045654296875, 0.89892578125, -1.265625, -1.2275390625, -0.8447265625, -0.60107421875, 0.6611328125, -2.865234375, -0.55029296875, -0.329833984375, 0.1114501953125, -0.258544921875, -1.2041015625, -0.2213134765625, -1.1884765625, 2.19140625, -1.3603515625, 0.62451171875, 1.2783203125, 0.11529541015625, 0.0675048828125, -0.296875, -1.19140625, 1.1923828125, -1.9365234375, 0.07415771484375, 0.32470703125, 0.126953125, 0.429931640625, 0.357421875, -0.421142578125, 0.4501953125, 2.59765625, -0.342041015625, -0.6767578125, 0.2366943359375, -1.611328125, 0.1695556640625, -0.40869140625, 2.9140625, -1.2548828125, -1.390625, -0.300537109375, -1.3642578125, -0.0723876953125, -0.4130859375, 1.515625, 1.0283203125, -0.306640625, 0.72802734375, -0.68017578125, -0.74658203125, -0.8212890625, 1.50390625, 1.0107421875, 0.05999755859375, 0.69970703125, -1.453125, 1.1689453125, 1.0986328125, 1.2255859375, 1.7939453125, 1.599609375, 1.9296875, 0.3603515625, 0.368408203125, -0.03302001953125, -0.405517578125, -0.259765625, -0.2176513671875, -0.322509765625, 0.01094818115234375, -0.9677734375, -1.6708984375, -0.4130859375, -0.65478515625, 0.81298828125, -0.0006923675537109375, -0.9375, 1.4365234375, 0.88232421875, -1.357421875, 0.1895751953125, -0.904296875, -1.2392578125, -0.215087890625, 0.387451171875, -0.3271484375, -0.375244140625, 1.146484375, 0.1656494140625, 0.47412109375, -2.404296875, -1.3310546875, -2.125, -1.7587890625, -0.75732421875, -0.465087890625, 0.79345703125, 0.10699462890625, 0.392333984375, -0.12408447265625, 0.2386474609375, 1.1162109375, 0.2117919921875, -0.56494140625, -0.6240234375, 0.60009765625, 0.437744140625, -2.12890625, -1.6171875, -1.25390625, 0.24609375, 0.708984375, 1.9228515625, 0.260986328125, 0.6044921875, -0.08343505859375, 0.390625, -0.53759765625, -0.65185546875, -0.28662109375, -0.84765625, -1.3203125, 0.402587890625, 0.29931640625, -1.265625, 0.14404296875, -1.025390625, -0.7802734375, 0.8203125, -1.1796875, -0.1456298828125, 0.6552734375, -0.248046875, 0.345458984375, -2.294921875, 0.916015625, -0.96435546875, -1.10546875, -1.1962890625, -0.5517578125, 0.369873046875, 1.6318359375, 1.6533203125, -0.0654296875, 2.33203125, 0.9140625, 1.61328125, -0.55810546875, 0.320068359375, -1.103515625, -0.08404541015625, -0.0098724365234375, 0.364013671875, -0.22998046875, 0.64013671875, -0.44873046875, -0.07635498046875, -1.5146484375, -0.5595703125, 0.275146484375, -1.3857421875, 0.04302978515625, 0.75341796875, -0.269775390625, -0.33203125, -1.2841796875, 0.1571044921875, 0.91015625, 1.646484375, 0.1024169921875, -1.046875, -1.7216796875, -0.0736083984375, 0.90869140625, 1.376953125, -0.61083984375, 0.6025390625, -0.57177734375, -0.06463623046875, -0.2218017578125, -0.31640625, 0.8369140625, 0.83447265625, 1.44921875, -0.7294921875, 1.0, 0.057647705078125, -1.9462890625, -0.7158203125, -0.46533203125, -2.158203125, 0.08624267578125, -0.2298583984375, 0.6962890625, -1.0146484375, 0.388427734375, -0.296142578125, 0.22998046875, 0.93115234375, 0.180419921875, 0.1322021484375, -0.1319580078125, -0.4638671875, -1.115234375, -0.2366943359375, -1.6025390625, 1.421875, 0.310302734375, -0.247314453125, -1.15234375, 1.759765625, 0.75341796875, 0.0213470458984375, -0.63525390625, -0.97412109375, -0.68115234375, -0.143798828125, 0.1109619140625, -0.72509765625, -0.99658203125, 0.300537109375, -1.1875, 1.40625, -0.939453125, -0.01010894775390625, 0.79345703125, 0.22216796875, 1.92578125, -1.4716796875, 0.412841796875, 1.3642578125, 0.2135009765625, -0.391845703125, 0.287353515625, -1.1376953125, -1.0498046875, -0.29052734375, 0.83203125, -0.0287017822265625, -0.75830078125, -0.91552734375, -1.951171875, -1.7353515625, 0.51513671875, -1.912109375, 0.32470703125, 0.04290771484375, 0.11993408203125, 0.5908203125, -0.71044921875, -0.86767578125, 1.3408203125, 1.3330078125, -1.033203125, -0.62646484375, 0.1231689453125, -0.55419921875, 0.67236328125, 0.72412109375, -0.466796875, 0.34326171875, -1.7197265625, 2.30078125, -1.1240234375, 0.7470703125, -0.344970703125, -1.1220703125, 0.146728515625, 0.235595703125, 0.248046875, 0.2081298828125, -0.215087890625, 0.7119140625, 0.1959228515625, -0.443603515625, -1.306640625, 0.638671875, 0.87255859375, -0.53564453125, 1.23828125, -0.8876953125, 0.17041015625, -0.95556640625, 0.442626953125, -0.2130126953125, 0.80224609375, 0.1519775390625, 0.13037109375, 0.66064453125, 2.388671875, 1.810546875, 0.62939453125, -0.66845703125, 1.048828125, 0.82568359375, 0.70703125, 1.1875, 1.17578125, -0.42919921875, -0.93994140625, 0.31591796875, 1.3125, 0.2373046875, -1.2861328125, -0.80078125, 1.001953125, 0.055694580078125, 0.6826171875, 0.277099609375, 0.182861328125, 0.0031871795654296875, 0.073974609375, -1.0419921875, 0.391357421875, -0.8427734375, -0.67138671875, -0.49560546875, -0.47509765625, 1.55859375, 0.5078125, -0.0924072265625, 0.318115234375, 1.75390625, 0.4482421875, 1.6650390625, 0.12152099609375, 0.2288818359375, 0.228759765625, -1.0556640625, 0.62939453125, 0.650390625, 1.451171875, -0.4052734375, 1.9208984375, -0.765625, -0.386474609375, 2.859375, 0.240478515625, -0.43359375, 2.05859375, -0.2203369140625, -1.6494140625, 1.1650390625, -0.92578125, -1.2138671875, 0.71484375, -0.354248046875, -1.005859375, 0.01209259033203125, -0.6923828125, 0.9990234375, 0.345703125, -0.65380859375, 0.5537109375, -0.96142578125, 0.6474609375, -1.0546875, -0.8271484375, 1.5771484375, -0.309326171875, 1.505859375, -0.79296875, -1.0390625, 1.025390625, 0.54345703125, -0.421630859375, 0.041046142578125, 0.53759765625, 1.6279296875, -2.091796875, 1.447265625, 0.279541015625, -0.129638671875, 0.097900390625, -0.351806640625, 2.203125, -2.12109375, -0.424560546875, 0.12890625, 0.2841796875, 0.12127685546875, -0.264404296875, -0.263427734375, 1.583984375, -1.4638671875, 0.2081298828125, 0.07415771484375, -0.1595458984375, 0.220458984375, 0.404052734375, -0.78076171875, 0.783203125, -1.8623046875, 0.29638671875, -1.0478515625, -0.34521484375, -0.156982421875, 1.70703125, -1.5908203125, -0.00463104248046875, -0.0210723876953125, 1.2802734375, -2.3671875, -0.69140625, -2.205078125, -0.454345703125, 1.2333984375, 0.01132965087890625, -0.7294921875, 1.9169921875, 0.7900390625, -1.5556640625, 0.04638671875, 0.16650390625, -0.27392578125, -0.5546875, -0.09173583984375, -0.292236328125, -0.3359375, 0.0714111328125, -0.379150390625, 1.015625, 0.39697265625, 1.634765625, 0.21435546875, -2.263671875, -0.73388671875, -0.190673828125, 0.1898193359375, -0.54833984375, 0.85498046875, 0.06085205078125, 0.7216796875, -0.619140625, 1.4755859375, 1.22265625, -0.9892578125, 0.60595703125, -1.9208984375, 0.0184478759765625, 1.1357421875, -0.459228515625, 0.64013671875, -1.5341796875, -0.6259765625, -0.5087890625, -1.00390625, -0.83251953125, -1.3388671875, -0.251953125, 0.00656890869140625, -1.638671875, -1.5556640625, 0.21142578125, 0.295166015625, 0.7568359375, 0.198486328125, -0.45361328125, 1.529296875, 0.5068359375, -0.283203125, 0.52978515625, 1.3837890625, -0.361083984375, -1.0517578125, 0.81103515625, 0.90576171875, -0.282470703125, -1.2451171875, -0.08184814453125, -0.75244140625, 1.8359375, 4.015625, 0.70166015625, 0.499755859375, -0.10345458984375, -1.67578125, 1.2626953125, -1.3701171875, 1.1875, -1.0888671875, -1.6865234375, -0.0975341796875, -2.095703125, 2.56640625, 0.4150390625, -1.3310546875, -0.210693359375, -0.08251953125, 0.271728515625, -0.28515625, -0.269287109375, 0.56884765625, -0.3642578125, 0.13720703125, -0.306640625, -1.26171875, 0.771484375, -0.77587890625, 0.1082763671875, 1.95703125, 0.27587890625, -0.04095458984375, -2.625, 0.1761474609375, -1.50390625, 0.57275390625, -0.86328125, -1.287109375, -0.97509765625, 0.366943359375, -1.88671875, -1.4140625, 0.2181396484375, -1.9443359375, -0.413818359375, -0.1220703125, 1.8642578125, 0.7578125, -0.52685546875, -1.3671875, 0.923828125, -0.493408203125, 0.94287109375, -0.76318359375, 1.31640625, -0.296875, -0.1971435546875, -0.274658203125, -0.20849609375, -0.061737060546875, 1.37890625, 1.6591796875, 0.60205078125, 0.484130859375, -0.9228515625, 1.5576171875, 0.192626953125, 0.3134765625, -0.1707763671875, 0.453125, 0.322265625, -0.0634765625, -0.1923828125, 2.146484375, -0.8193359375, -1.2744140625, 0.0709228515625, -1.1708984375, 0.89794921875, 0.0045623779296875, 1.9033203125, 0.416259765625, -1.0234375, 0.2037353515625, -1.0517578125, -0.1822509765625, 0.75927734375, 0.76611328125, -0.172119140625, -0.450927734375, 0.61083984375, 0.1363525390625, 1.486328125, 0.53271484375, 1.23046875, 1.5, -0.07354736328125, -0.1767578125, -1.41015625, 0.10528564453125, 1.451171875, 2.703125, 1.064453125, -0.853515625, 1.109375, -2.4296875, -0.96337890625, 1.54296875, 1.3037109375, -0.36572265625, 0.315185546875, 0.238525390625, -0.748046875, 0.345703125, -0.487060546875, 0.0019664764404296875, -1.5498046875, 0.381591796875, 0.1502685546875, -0.05255126953125, -0.77099609375, -1.736328125, 0.27783203125, -0.00028896331787109375, -0.81494140625, -0.264404296875, -0.966796875, 0.10400390625, 0.47802734375, 0.0660400390625, -0.2154541015625, -0.43798828125, -0.415283203125, -0.050445556640625, -0.185791015625, 0.28173828125, -1.666015625, -1.1025390625, -0.322998046875, 0.7421875, 0.50341796875, -1.76953125, 1.1748046875, 0.6865234375, -1.10546875, 1.564453125, -0.357421875, 2.001953125, -1.6005859375, -0.55712890625, 2.244140625, 1.87890625, -1.5068359375, -0.139892578125, -1.142578125, -0.08404541015625, 0.806640625, 0.11407470703125, -1.87890625, -1.7158203125, -1.4765625, 1.2060546875, 0.81201171875, 0.79931640625, -1.6328125, -0.728515625, 2.458984375, -0.72216796875, 0.1697998046875, -1.21484375, 0.52880859375, -0.68798828125, -0.76953125, -1.1064453125, 0.22021484375, 1.7861328125, -0.211181640625, -2.3125, -0.34912109375, 0.9384765625, 0.3017578125, -1.4111328125, -0.074462890625, 0.28466796875, 1.1728515625, -0.12017822265625, -0.37255859375, -2.052734375, -0.3671875, 0.59228515625, -0.58544921875, 0.284912109375, 1.333984375, -1.81640625, -0.07647705078125, -0.10888671875, -1.9091796875, 0.34423828125, 0.67529296875, 1.041015625, -0.306884765625, 0.50537109375, -2.693359375, -0.86376953125, -0.69580078125, -0.1572265625, -0.5849609375, -0.09674072265625, 0.2183837890625, 1.28125, -0.295654296875, -1.1064453125, 0.7080078125, -1.046875, -0.70068359375, -0.05426025390625, -0.671875, 0.1512451171875, -1.26953125, -0.0142974853515625, -0.466796875, -0.07562255859375, -0.031768798828125, -0.323486328125, -0.2120361328125, 0.0711669921875, -0.69677734375, -1.779296875, -1.921875, -0.90771484375, 0.73876953125, 0.62646484375, 0.7236328125, 0.69091796875, 0.040924072265625, 0.51806640625, 0.52685546875, 0.3837890625, -1.349609375, -0.87646484375, 1.0625, 0.047393798828125, 0.1712646484375, -0.82861328125, -0.1363525390625, 2.177734375, -1.0380859375, 0.6376953125, 0.49853515625, 0.46630859375, -0.1949462890625, 0.65966796875, 0.377197265625, -0.748046875, 0.060455322265625, 0.92138671875, -0.55615234375, 0.57177734375, -1.2158203125, 0.68603515625, 0.385986328125, 0.74169921875, 0.7744140625, 2.017578125, -1.5029296875, -0.0102081298828125, 1.9560546875, -0.41650390625, -1.04296875, -1.9345703125, 0.377197265625, -0.77197265625, 1.072265625, -0.71484375, 0.348876953125, 0.583984375, 0.34521484375, -1.0810546875, 0.02532958984375, -0.259765625, -1.416015625, 0.1199951171875, 0.047119140625, 0.253662109375, -0.57373046875, 2.119140625, -1.3037109375, 0.603515625, 0.3916015625, -0.65234375, 1.4521484375, -0.281494140625, 0.5888671875, 0.38232421875, -1.1259765625, -0.8779296875, -0.91748046875, -1.1279296875, -0.27783203125, -0.323974609375, -0.264404296875, -0.2099609375, -0.87353515625, 0.1800537109375, 0.279052734375, -0.287353515625, 0.5107421875, -1.431640625, -2.1953125, 0.377197265625, -1.64453125, 0.178955078125, -0.82177734375, -1.9697265625, -0.56689453125, -0.92138671875, 0.458740234375, 0.53369140625, -0.084228515625, -1.4638671875, -0.181884765625, -0.0015001296997070312, 1.1533203125, -1.388671875, -0.759765625, -0.5029296875, -1.1005859375, -0.279541015625, 0.54736328125, 0.53271484375, -0.142578125, 0.5224609375, -0.09356689453125, -2.04296875, 0.125732421875, 0.340087890625, 0.94091796875, -1.521484375, 0.2259521484375, -0.1619873046875, 0.21533203125, 0.2193603515625, -1.5234375, 0.55029296875, 1.3359375, 0.55908203125, -0.521484375, 0.619140625, 1.2724609375, -0.5380859375, -1.0234375, 0.2164306640625, -0.2088623046875, 0.38134765625, -0.2225341796875, -0.45263671875, 0.59521484375, -0.296630859375, -0.6103515625, -1.0634765625, 0.61962890625, -1.7666015625, 0.24755859375, -0.424560546875, -1.291015625, -1.564453125, 1.0732421875, 0.642578125, -1.1728515625, 1.6220703125, -0.32958984375, 0.30859375, 0.74951171875, -0.93359375, -0.1126708984375, -1.5302734375, 0.65966796875, 0.9951171875, -0.39453125, -0.5830078125, 0.44287109375, 0.431884765625, 0.73291015625, -0.72265625, -1.3818359375, 1.7041015625, -0.26123046875, 0.7294921875, 0.495849609375, -0.2066650390625, -0.2044677734375, 0.73779296875, -0.52099609375, -0.05023193359375, -0.73046875, 0.560546875, 0.8154296875, 0.814453125, -0.409423828125, -0.7890625, -1.3251953125, 0.39794921875, 1.3837890625, 1.8896484375, -0.51904296875, 2.1875, 0.5859375, -0.47509765625, 1.3125, -1.0712890625, 0.5830078125, 0.11822509765625, -0.6181640625, 0.23291015625, 0.444580078125, -0.97314453125, -0.63525390625, 0.6611328125, 0.900390625, 0.287109375, 0.260009765625, 1.380859375, -1.3349609375, -0.300537109375, 0.8046875, 0.208740234375, 0.12286376953125, -0.5654296875, 0.83447265625, 2.189453125, 1.1240234375, 0.7431640625, -0.91259765625, 1.869140625, 0.8798828125, 0.1702880859375, 0.368896484375, -0.69287109375, 1.1201171875, -1.5966796875, -0.54638671875, 0.39404296875, -0.62841796875, -0.9404296875, 3.703125, 0.5224609375, -0.1429443359375, -0.0810546875, -0.466064453125, -0.7880859375, -0.93359375, 0.4892578125, 1.078125, 0.63232421875, -0.6015625, -0.50341796875, -0.49267578125, 0.440185546875, -0.6513671875, 0.794921875, -0.5693359375, -0.6416015625, 0.2724609375, -0.01482391357421875, -1.23828125, 0.49169921875, -0.390380859375, 0.919921875, -0.36279296875, -2.349609375, -0.96826171875, 0.56640625, -1.9482421875, 0.412841796875, 0.1492919921875, 0.736328125, 1.3427734375, -0.2099609375, 0.380859375, 1.10546875, 2.08984375, 0.12646484375, 1.2763671875, -0.427001953125, -1.7607421875, 0.8486328125, -0.1654052734375, -0.092529296875, 1.2470703125, -0.2188720703125, 0.84619140625, -0.85205078125, -0.6669921875, 0.82177734375, 1.52734375, -0.1676025390625, 0.9150390625, -2.123046875, 1.3544921875, -0.90283203125, -1.2265625, 0.255126953125, 1.373046875, 0.368408203125, 0.4931640625, -0.406982421875, 2.533203125, -1.4326171875, -0.376708984375, -1.0341796875, 0.63720703125, 0.609375, -1.8603515625, 0.62744140625, 0.818359375, -0.93212890625, 1.3681640625, 0.54443359375, -0.4873046875, 0.70654296875, 1.3544921875, 0.11712646484375, -0.471923828125, 0.12274169921875, -0.009674072265625, -0.65673828125, 0.73486328125, 0.53662109375, 0.138671875, 1.47265625, 1.2421875, 0.60400390625, -1.5830078125, -3.015625, 1.3271484375, -0.28857421875, -0.247314453125, -0.60546875, -0.759765625, 0.52587890625, 0.035552978515625, -0.05169677734375, -1.6669921875, 0.43359375, -1.3857421875, 0.2496337890625, 0.478271484375, -0.52978515625, 0.77783203125, 1.1708984375, 0.51513671875, 0.7783203125, 0.724609375, 0.82421875, -0.0195159912109375, -0.1541748046875, -0.183837890625, -0.1011962890625, 0.51513671875, 0.904296875, -0.196044921875, 2.09375, 1.5869140625, 0.89990234375, 0.57666015625, 0.86767578125, 0.445068359375, -0.740234375, -0.10247802734375, -0.0684814453125, 2.36328125, 1.337890625, -0.89697265625, 0.352294921875, -1.06640625, 1.544921875, 1.2080078125, 1.0595703125, 0.07379150390625, 0.59326171875, -0.78857421875, -1.6767578125, -1.1845703125, 0.6171875, 0.35546875, 0.35400390625, -0.4140625, 0.399169921875, 0.419677734375, 1.095703125, 0.481689453125, 0.95361328125, -1.3515625, 1.0498046875, 1.1044921875, 0.7998046875, -0.91943359375, 0.93994140625, 1.466796875, -0.1422119140625, 0.72021484375, -0.75341796875, 0.08721923828125, -1.84765625, 0.80126953125, 1.056640625, 0.242431640625, 1.1123046875, -0.82275390625, 1.2919921875, -1.1572265625, 0.63525390625, 2.23828125, 0.0300445556640625, 0.068603515625, -2.037109375, -0.4453125, 0.0226593017578125, -0.312744140625, -0.06854248046875, 0.222412109375, -0.86181640625, 0.2685546875, -0.64111328125, -1.6826171875, 0.77734375, -0.61474609375, 0.400390625, -0.59326171875, 1.234375, -0.58837890625, -0.4375, -0.489990234375, -0.1439208984375, 0.44677734375, 0.0892333984375, 0.80712890625, 0.58837890625, 0.2296142578125, 0.53125, 0.6455078125, -0.701171875, -0.01134490966796875, 0.55322265625, 0.6259765625, -2.771484375, 1.80078125, 1.3681640625, 1.2294921875, -0.6318359375, -0.33984375, 0.62158203125, 0.436767578125, 0.60546875, 0.70654296875, -0.9365234375, 1.025390625, -0.2381591796875, -0.54638671875, 0.84716796875, -1.3173828125, -0.73681640625, -0.44580078125, -1.052734375, -0.0523681640625, -0.6572265625, -0.6103515625, -1.2998046875, -0.327880859375, -0.11083984375, -0.9150390625, 0.455810546875, -1.1279296875, -0.59130859375, -0.189697265625, 0.341796875, 0.494384765625, -0.35693359375, 0.77392578125, -1.8427734375, -0.298095703125, 1.29296875, 1.125, 1.736328125, -0.35498046875, 0.8984375, 1.2958984375, -0.60693359375, -0.12493896484375, 0.041290283203125, -0.252685546875, 1.5361328125, -0.5400390625, -0.2822265625, -0.5322265625, 0.8310546875, 0.2266845703125, 0.56591796875, 0.52294921875, 0.92041015625, 1.33203125, 1.1962890625, 0.5146484375, -0.95849609375, 1.7099609375, -1.267578125, 1.1494140625, 0.49853515625, -0.615234375, -0.779296875, -0.615234375, -0.043426513671875, -0.01085662841796875, 0.33837890625, -0.83984375, 1.5087890625, 0.45556640625, 0.0743408203125, 0.319580078125, 1.23828125, 1.00390625, -1.3271484375, 0.806640625, -1.0087890625, -0.3017578125, -0.137451171875, -1.322265625, -0.68115234375, 0.4033203125, 0.0288848876953125, 0.203125, 0.63671875, 1.0224609375, -0.736328125, -0.35205078125, -0.344970703125, -2.568359375, -0.57470703125, -0.37451171875, -0.85498046875, -0.54931640625, 0.1776123046875, -2.107421875, 1.0029296875, 0.1746826171875, -0.1307373046875, 0.1295166015625, -1.6083984375, -1.251953125, 0.01947021484375, -1.580078125, 2.51171875, -1.21875, 0.59423828125, 0.278564453125, 0.1510009765625, -0.414306640625, 1.6796875, 0.947265625, -1.7431640625, 0.1419677734375, 1.42578125, 1.0654296875, 0.2059326171875, -1.41015625, 2.533203125, -1.330078125, -0.798828125, 1.169921875, -0.63134765625, -1.62109375, -1.1123046875, -1.5078125, 1.19140625, -0.36865234375, 1.251953125, 0.4765625, 0.869140625, 1.7890625, -0.73779296875, 0.61669921875, -0.146484375, -0.007419586181640625, -0.12176513671875, -0.06793212890625, -1.1982421875, 1.7578125, -0.454345703125, -0.0214385986328125, 1.98046875, 0.69775390625, 1.1259765625, 1.4111328125, 0.5908203125, 0.29443359375, 1.30859375, 1.2099609375, 1.5478515625, -0.7939453125, 0.1534423828125, -0.1591796875, -1.5888671875, -0.60986328125, -1.0830078125, 1.546875, -0.9765625, -1.287109375, -0.2861328125, -0.41162109375, -0.6630859375, 0.50341796875, -0.7275390625, -0.970703125, -2.044921875, 0.193115234375, -1.1533203125, -1.2255859375, -0.214599609375, 1.380859375, 0.0958251953125, 1.2060546875, -0.97216796875, 1.05078125, 1.044921875, -0.8330078125, 0.9521484375, -0.91650390625, -0.2196044921875, -0.086669921875, -1.171875, 0.22216796875, 0.346435546875, 0.4609375, -0.486328125, 0.2093505859375, 0.03521728515625, 0.013885498046875, 0.122314453125, -0.2496337890625, 0.91015625, -0.57080078125, -0.18798828125, -0.2003173828125, -0.285888671875, 0.1976318359375, -0.9326171875, -0.446044921875, 0.260009765625, 0.1737060546875, 0.13427734375, -0.373291015625, -0.607421875, 0.26416015625, -0.57568359375, -1.1982421875, 0.139892578125, -0.6162109375, -3.06640625, -0.55615234375, 0.372314453125, 0.432373046875, -0.95458984375, -0.30859375, 0.26318359375, 0.259765625, -1.1796875, 2.35546875, -1.3896484375, 0.269287109375, 1.197265625, -0.1685791015625, -1.486328125, -0.46826171875, 2.04296875, 0.87841796875, 0.281005859375, -0.387939453125, -0.09210205078125, -1.1904296875, 0.615234375, 0.9228515625, -0.48828125, -0.77880859375, 0.4970703125, 1.58984375, 1.0546875, 0.272705078125, -0.232177734375, 0.484375, 0.52001953125, 0.58837890625, -1.2119140625, -0.576171875, -0.197509765625, -2.482421875, -1.080078125, -2.072265625, 2.603515625, 1.8779296875, -0.75537109375, 0.41259765625, -0.3251953125, 1.0322265625, -0.94287109375, -0.78955078125, -0.299072265625, -0.2366943359375, -0.06024169921875, 0.0948486328125, -0.08294677734375, 0.451171875, -1.5576171875, 0.349853515625, 0.36279296875, 1.2890625, 1.4912109375, 0.4501953125, 1.2109375, -1.724609375, 0.12213134765625, -1.3603515625, -0.03662109375, -0.48046875, 0.06494140625, 0.9716796875, -0.318115234375, -1.00390625, -0.7470703125, 1.591796875, -0.351806640625, 0.032684326171875, -0.4853515625, -0.84765625, 0.24755859375, -1.3271484375, -0.275146484375, 0.04071044921875, -1.5810546875, 0.235107421875, -1.36328125, 0.6240234375, -0.47265625, 0.385009765625, -0.73388671875, 2.38671875, 2.013671875, -0.402099609375, 0.9560546875, 1.154296875, -0.08599853515625, -0.88232421875, 0.48681640625, -1.0947265625, 0.63232421875, -0.411376953125, -1.943359375, 0.10430908203125, 3.177734375, 0.366943359375, -0.98828125, -0.1461181640625, -0.157958984375, 0.923828125, -0.60107421875, 0.15673828125, 2.068359375, 1.6923828125, -0.424560546875, 0.10882568359375, 0.65087890625, 0.31298828125, 0.16162109375, -0.054718017578125, 0.5078125, 1.0732421875, 1.1328125, -0.98046875, -0.8408203125, -0.31494140625, -1.234375, 0.413818359375, 1.2822265625, -0.75, 2.169921875, -0.93359375, -0.0027408599853515625, -0.57470703125, 0.421875, 0.11065673828125, 0.310791015625, -0.62353515625, 2.236328125, -1.310546875, -1.7353515625, -0.358154296875, 0.5107421875, 1.0732421875, 3.189453125, -0.5458984375, -0.474365234375, 0.483154296875, -0.87451171875, 0.734375, 2.013671875, 0.1300048828125, 1.552734375, 0.1663818359375, 0.158203125, 1.6220703125, 2.384765625, -0.55224609375, 0.453125, 0.7900390625, 2.986328125, 1.3896484375, -1.3564453125, -1.1904296875, -1.55859375, 0.58447265625, 1.171875, -0.315673828125, -0.8515625, -0.53173828125, 1.248046875, 2.091796875, -0.37890625, 0.08331298828125, 0.5205078125, -0.1544189453125, 0.35400390625, 0.7275390625, -0.68798828125, -0.6494140625, 0.336181640625, -0.373291015625, -0.58203125, 1.138671875, 1.6171875, 0.285888671875, 0.76806640625, -0.2060546875, -0.1737060546875, -0.39794921875, -0.86376953125, -1.12109375, 0.83251953125, 0.47216796875, 0.61962890625, 1.443359375, 0.206787109375, -0.5888671875, -0.2235107421875, 0.4921875, 2.50390625, 1.4609375, 1.1962890625, 1.6689453125, -0.1912841796875, -0.95556640625, 1.0546875, 0.0161895751953125, 0.3330078125, 0.3330078125, -2.169921875, 0.1697998046875, 1.162109375, 0.128662109375, 0.873046875, 0.51513671875, -2.73046875, 0.48583984375, -1.4462890625, -0.38818359375, 0.11920166015625, -1.701171875, -0.272705078125, 0.08599853515625, 0.8701171875, 1.1318359375, 0.78759765625, 0.44970703125, -1.681640625, -1.3408203125, 1.048828125, -0.80859375, -0.89453125, -0.1790771484375, -0.71630859375, -0.295166015625, -0.361083984375, 0.68310546875, -0.5361328125, -0.182373046875, 1.3349609375, 0.319580078125, -1.9990234375, -0.716796875, 0.187255859375, -1.3984375, -0.293212890625, 0.5888671875, -0.5166015625, 1.2822265625, 0.8271484375, -0.85546875, 0.428466796875, -0.59912109375, 0.123046875, 1.49609375, -0.62890625, -0.81005859375, 0.315185546875, -0.998046875, -1.2412109375, -0.276611328125, 1.126953125, -0.0318603515625, -1.427734375, -1.3310546875, 0.34375, 1.19140625, -0.93115234375, 0.90380859375, -2.16015625, -0.72216796875, -1.10546875, 0.7958984375, 1.318359375, -0.432861328125, 1.119140625, 0.58056640625, 0.587890625, -0.453857421875, 0.171875, 0.64501953125, 1.0576171875, 0.8310546875, 0.4931640625, 0.44921875, 0.65087890625, 0.232177734375, 0.05340576171875, -0.5693359375, 0.285400390625, 0.6435546875, -0.385009765625, 1.650390625, -0.38525390625, 0.56103515625, 2.55859375, -2.00390625, -0.266845703125, 0.312255859375, 0.7138671875, 0.50390625, -0.1854248046875, -0.626953125, -0.818359375, 1.9990234375, -0.1175537109375, 0.712890625, 1.3779296875, 0.053253173828125, -1.3271484375, 0.869140625, -0.7490234375, 1.091796875, 0.85693359375, 0.27734375, -0.8955078125, -0.218017578125, -0.8984375, 0.0189971923828125, -0.3720703125, 0.3427734375, 0.271728515625, 0.345947265625, 0.37744140625, -0.310791015625, 0.68798828125, 0.95703125, 1.1162109375, 0.65380859375, -0.86767578125, 0.1630859375, 0.62060546875, 1.16796875, -0.822265625, -0.7216796875, 0.2222900390625, 0.82177734375, -0.7451171875, -0.9453125, 0.5, 1.3193359375, -1.2939453125, 1.4462890625, -1.2001953125, -0.06988525390625, 0.70263671875, -0.0489501953125, 0.76513671875, -0.07958984375, 1.025390625, 0.462646484375, 1.7529296875, 1.123046875, 1.5087890625, 1.3583984375, 1.3251953125, -1.009765625, -0.76123046875, 1.3330078125, -0.14111328125, 0.5458984375, -0.642578125, -0.291259765625, 0.474365234375, 0.08172607421875, -0.367919921875, -1.51953125, 0.069091796875, 0.320068359375, 0.630859375, 0.57470703125, 0.66259765625, 1.1171875, 0.00846099853515625, -0.1297607421875, 1.6240234375, -0.326171875, 0.12158203125, 0.06658935546875, 1.6484375, -0.364013671875, 0.0311126708984375, -0.50048828125, -1.935546875, -2.19921875, -0.1407470703125, 0.4052734375, -0.74169921875, -0.08111572265625, -0.63818359375, 0.465087890625, 0.265625, -0.95751953125, -0.1265869140625, 1.5556640625, -0.46875, 0.65185546875, -1.28125, 0.662109375, 0.5576171875, -0.4521484375, -0.98681640625, -1.6318359375, -0.75732421875, -3.146484375, -1.4072265625, -1.4541015625, 0.229248046875, -0.1690673828125, 0.654296875, -0.300537109375, 0.250732421875, 0.1131591796875, -0.677734375, -0.363525390625, -0.066650390625, 0.52099609375, -0.58642578125, -0.12744140625, 0.63720703125, -1.9521484375, -1.1767578125, 0.47509765625, -0.288818359375, -0.1556396484375, 0.1187744140625, 1.8642578125, -0.08447265625, -1.6669921875, -0.282958984375, -0.78076171875, 0.082763671875, -0.0882568359375, -0.3740234375, -0.4580078125, 1.388671875, -0.0002789497375488281, 0.6865234375, -0.35546875, -0.84228515625, 0.01190948486328125, 0.076416015625, -0.21826171875, 0.55517578125, 0.031982421875, 0.45556640625, 0.82568359375, -0.587890625, -0.5458984375, 0.05712890625, 1.140625, -1.009765625, 0.494384765625, -1.4921875, -0.96240234375, 0.6005859375, -1.021484375, 0.5068359375, 1.068359375, -1.3466796875, -0.211669921875, 1.1572265625, 0.74462890625, 0.135986328125, -2.373046875, 0.219970703125, -0.054351806640625, 0.184814453125, -1.736328125, -0.92919921875, 2.5546875, -0.5400390625, 0.72705078125, -0.64404296875, 0.787109375, -0.90771484375, 0.60595703125, -0.60595703125, -1.5439453125, 1.822265625, 1.4560546875, 1.1162109375, -1.6142578125, -0.488525390625, -0.29541015625, 0.055816650390625, -0.43212890625, 0.1680908203125, -0.44921875, -1.13671875, 0.7841796875, 1.41796875, 0.76611328125, 0.73046875, 1.0205078125, 0.84619140625, 0.54443359375, -0.8017578125, -0.379150390625, -0.435302734375, -0.07440185546875, 1.1064453125, 0.24560546875, -0.014678955078125, 1.2392578125, -1.53515625, 0.00278472900390625, -1.763671875, -0.1739501953125, -0.184814453125, 0.53125, -0.207763671875, -0.28759765625, -0.44482421875, -1.123046875, -1.7119140625, -0.9228515625, -0.8427734375, -0.51708984375, 0.120849609375, 1.4189453125, -0.818359375, 0.425048828125, -0.229736328125, 1.0078125, 0.1337890625, 1.4150390625, -1.5439453125, -1.4794921875, -0.01248931884765625, -1.380859375, 0.0041656494140625, -1.3134765625, 0.03399658203125, 2.0234375, -2.05078125, 0.27685546875, 0.387451171875, -0.3896484375, 1.09765625, -0.05987548828125, -0.953125, 0.29931640625, -0.51953125, -0.285888671875, 0.10430908203125, 0.448486328125, -0.97216796875, -1.2197265625, 0.082763671875, 1.4130859375, -0.39794921875, -1.6318359375, -1.4130859375, 0.98974609375, -0.78369140625, -0.1431884765625, -1.05078125, -0.304443359375, -0.5810546875, -0.464599609375, 0.430419921875, -1.9150390625, 0.0887451171875, 1.1328125, -0.8525390625, -1.3271484375, -0.93603515625, 0.9462890625, -0.0595703125, 0.50244140625, -1.2294921875, 1.5478515625, 0.398193359375, -1.205078125, 0.75927734375, -0.7158203125, 0.3505859375, -1.1923828125, -0.213623046875, 1.7138671875, 0.97314453125, 1.2109375, 0.5947265625, -0.4560546875, -2.0546875, 0.3017578125, 0.3935546875, -0.75537109375, 0.88037109375, 0.5634765625, -0.5068359375, -0.83349609375, 0.0804443359375, -1.1943359375, 0.509765625, 0.38525390625, 0.250244140625, -1.115234375, 0.896484375, -2.98828125, -1.2587890625, 0.1976318359375, 0.119873046875, -0.51806640625, -0.84765625, -0.0123138427734375, -0.260498046875, 0.4619140625, -0.407958984375, 0.2103271484375, -0.1331787109375, 0.9140625, -0.6796875, -0.64453125, 0.05206298828125, 0.23486328125, 0.60546875, -0.79150390625, 2.017578125, 0.143310546875, 0.0159912109375, 0.2626953125, -0.24853515625, -1.4248046875, -1.10546875, -0.1788330078125, -0.935546875, 0.38525390625, 0.0738525390625, -1.0400390625, 0.73876953125, -0.83447265625, -0.21337890625, -0.40283203125, 1.5634765625, -1.015625, -0.9970703125, 0.378173828125, -0.07403564453125, -2.53125, -0.272216796875, 0.716796875, -0.90966796875, -1.7607421875, 1.173828125, -1.396484375, 2.029296875, -0.7509765625, -1.7138671875, 0.04669189453125, -0.287841796875, 1.3662109375, -1.2421875, -0.135498046875, -0.7939453125, 0.43798828125, 1.3017578125, 1.2265625, 1.693359375, -0.047698974609375, -1.142578125, 1.501953125, -0.61572265625, -0.13671875, -1.9189453125, 1.5390625, 0.0946044921875, 0.159912109375, -0.68359375, -0.393798828125, 0.494873046875, 0.3447265625, -0.28564453125, -0.141357421875, 1.2958984375, -0.5888671875, 0.428955078125, -0.395751953125, -1.390625, 0.047607421875, 1.2412109375, 0.708984375, 0.96240234375, 0.375244140625, 1.8212890625, 0.300048828125, 1.3779296875, 0.2315673828125, 1.748046875, -2.306640625, 1.517578125, 0.919921875, 0.6982421875, -1.1220703125, 2.1171875, -0.06524658203125, -0.84130859375, -2.08984375, -0.1063232421875, -1.173828125, -0.26611328125, 1.0810546875, -0.9765625, 0.46630859375, -0.5859375, 0.1080322265625, 1.6865234375, -0.79833984375, -0.303466796875, 1.1669921875, 0.402587890625, 0.681640625, -1.703125, -0.476318359375, 0.222900390625, -1.1357421875, 1.16015625, -0.63037109375, 1.5810546875, 0.041351318359375, 0.410888671875, 0.140380859375, -0.6884765625, 0.295654296875, -0.24462890625, -1.447265625, -0.2357177734375, 2.17578125, 0.09979248046875, 2.603515625, 0.5947265625, 0.260009765625, 1.6240234375, 0.2900390625, 0.0266265869140625, 1.603515625, 2.263671875, 1.96875, 2.6796875, -0.30712890625, 0.289794921875, -0.9150390625, 0.40771484375, -0.2130126953125, -0.479248046875, 0.0875244140625, 0.35888671875, -0.472412109375, -0.669921875, 0.3291015625, -1.2275390625, 0.0616455078125, -0.408203125, 0.83544921875, -0.6494140625, -0.003292083740234375, -0.11212158203125, -1.8935546875, -0.1177978515625, 1.3076171875, 0.19677734375, -1.7529296875, -0.370361328125, 0.5693359375, -0.253662109375, 0.349853515625, 2.650390625, 0.52587890625, -0.235595703125, -0.255859375, 0.07159423828125, -0.217529296875, 0.5595703125, -0.339111328125, -0.439453125, 0.6767578125, -1.072265625, 0.065185546875, 2.455078125, -1.2265625, -0.71044921875, -2.349609375, -0.98193359375, 0.44091796875, 1.130859375, -1.2412109375, 0.8740234375, -0.60205078125, -0.5634765625, -0.4384765625, -2.1484375, 2.14453125, -0.1546630859375, 0.900390625, -1.2099609375, -0.51904296875, 0.41357421875, -0.06353759765625, 1.3095703125, 1.765625, -0.95849609375, 1.197265625, 1.505859375, -0.8427734375, 1.7958984375, 1.255859375, -0.0556640625, -0.03094482421875, 0.67333984375, -0.1837158203125, 0.4189453125, 0.29296875, 0.73046875, 0.304931640625, -0.304443359375, 0.921875, 0.1929931640625, -2.046875, 0.29150390625, 0.053466796875, -0.67138671875, -0.44189453125, 0.31494140625, -2.1328125, -0.734375, -1.0927734375, 0.1190185546875, -0.263427734375, 0.412109375, -1.646484375, -0.465576171875, 1.912109375, 1.22265625, -1.6103515625, 0.220703125, 0.76953125, -0.81982421875, -0.48291015625, 0.66845703125, -0.039947509765625, 1.7265625, -2.80078125, 1.033203125, -1.1083984375, -1.8271484375, -1.3515625, -0.64306640625, -1.4794921875, 0.356689453125, -1.2744140625, 0.2117919921875, -0.2044677734375, 1.71484375, -0.90283203125, -1.7373046875, 0.5478515625, -1.283203125, 0.2471923828125, -0.12347412109375, 0.35107421875, 0.87841796875, 2.07421875, 0.287109375, 0.64501953125, -1.43359375, -0.53466796875, -1.5205078125, -0.6650390625, 0.362060546875, 1.2734375, 0.73974609375, 0.1978759765625, 0.302978515625, 0.42431640625, 1.05078125, 0.061370849609375, -0.1907958984375, -2.263671875, -1.2578125, -0.599609375, 1.4619140625, 0.075439453125, -0.5517578125, 0.1978759765625, 1.3720703125, -0.3037109375, 0.328125, 1.0009765625, -0.80126953125, 0.453125, 1.4765625, -0.22705078125, -1.9716796875, -1.7822265625, 0.90087890625, -0.07000732421875, 0.0276641845703125, -0.8759765625, 0.79931640625, 2.232421875, 0.8974609375, 1.3271484375, 0.0205230712890625, 0.2724609375, -1.1357421875, 3.78515625, 0.734375, -0.7939453125, -2.12109375, -1.01171875, -0.8046875, -0.5869140625, -2.16015625, 0.459716796875, -0.76220703125, -1.0654296875, 0.134521484375, 1.2529296875, 0.356689453125, 0.438232421875, 0.59521484375, -0.2088623046875, -1.92578125, -0.55517578125, -1.2001953125, -2.421875, -0.188720703125, -1.447265625, 0.0029468536376953125, -0.396240234375, 1.4892578125, -1.7568359375, -1.017578125, 0.272216796875, 0.71484375, 0.0258941650390625, 0.5634765625, 0.240966796875, -0.6240234375, -0.131591796875, -1.275390625, -1.537109375, -2.009765625, 1.0986328125, 0.6328125, -2.228515625, -0.52392578125, -0.95166015625, 1.064453125, -0.96142578125, -0.4794921875, 1.376953125, -1.5185546875, 1.99609375, -0.685546875, -1.6689453125, -1.7109375, -0.796875, 0.453857421875, -1.2197265625, 1.248046875, -0.388427734375, 0.2261962890625, -1.28125, -1.3720703125, -1.1396484375, -0.2325439453125, -0.728515625, -0.073974609375, -0.2039794921875, -0.115478515625, -1.435546875, 0.91796875, -0.07281494140625, 1.638671875, 1.47265625, -0.025238037109375, 0.9072265625, 0.30224609375, -0.97802734375, 0.198974609375, -0.26025390625, 0.1434326171875, 0.5732421875, 1.833984375, 0.6767578125, 0.88330078125, -1.5400390625, -0.09881591796875, -1.1484375, 1.33203125, 0.35693359375, -1.7265625, 0.497802734375, 1.2197265625, -0.035797119140625, -0.474365234375, 0.315673828125, 1.3291015625, 0.435546875, -1.724609375, -0.626953125, 0.61767578125, -1.029296875, -0.76953125, 0.6728515625, 1.0859375, -1.1728515625, 0.50537109375, -0.494873046875, -0.93896484375, 1.5126953125, 0.258544921875, 0.319580078125, 0.5166015625, 1.0390625, -0.09881591796875, 0.300048828125, -0.0728759765625, 0.529296875, 1.44921875, 0.818359375, 0.6328125, -2.5703125, -1.494140625, -0.80810546875, -0.62158203125, -1.0537109375, 0.38427734375, -0.7197265625, -0.159423828125, -0.492431640625, 0.9609375, 0.7841796875, -0.5693359375, 0.74072265625, -1.650390625, -0.1695556640625, 0.7470703125, -0.100341796875, -1.5341796875, 0.67529296875, -0.48486328125, -1.16015625, -0.90673828125, -1.859375, 0.043304443359375, -0.0030975341796875, -1.5927734375, -0.41943359375, 0.86767578125, -0.578125, -0.80615234375, -0.46533203125, 3.02734375, -1.142578125, -2.16015625, -0.826171875, -0.638671875, -0.626953125, 0.479736328125, -0.65869140625, 0.3271484375, 2.60546875, -0.91015625, 1.4462890625, 0.1539306640625, 1.3603515625, 0.36279296875, -0.7216796875, -1.2861328125, 1.04296875, -0.28955078125, 0.08441162109375, 0.06683349609375, 1.9462890625, -0.43359375, -0.262939453125, 0.100830078125, -0.046966552734375, -0.7294921875, -0.87744140625, -0.03802490234375, 0.1343994140625, -1.2138671875, 0.57177734375, 0.0743408203125, 2.0078125, 0.60205078125, -1.6728515625, -2.78125, -0.308349609375, 0.6767578125, 0.05517578125, -1.943359375, -1.38671875, -0.151611328125, -0.461181640625, -1.3203125, 1.5693359375, -0.2259521484375, -1.3095703125, -0.697265625, -0.93310546875, 0.74267578125, -1.103515625, -2.57421875, 0.34326171875, 0.34130859375, 1.8779296875, 0.1888427734375, -1.7958984375, -1.544921875, -2.255859375, -0.359130859375, 0.814453125, 0.30859375, -0.884765625, 0.03839111328125, 0.28369140625, 0.383056640625, -0.85791015625, -0.63818359375, -1.0888671875, -1.2470703125, -1.2001953125, 1.525390625, 0.06878662109375, -1.2802734375, 1.099609375, -0.479248046875, 0.8857421875, -1.2197265625, -2.185546875, 0.37548828125, 0.0789794921875, -1.30078125, 0.1024169921875, 0.376220703125, 0.039093017578125, 0.26123046875, 1.5078125, 0.810546875, 0.5244140625, -0.87060546875, -0.67919921875, -0.9853515625, 0.4677734375, -0.52294921875, -0.19677734375, 0.2486572265625, -0.625, 0.599609375, 0.61767578125, 0.290771484375, 3.18359375, -2.04296875, 0.5732421875, 1.5576171875, -0.6728515625, -0.416259765625, 0.298583984375, -1.0322265625, -0.09625244140625, -0.0867919921875, 0.33984375, 0.767578125, 0.52734375, 0.9404296875, 0.052520751953125, 0.8232421875, -0.1611328125, -1.7529296875, -0.299072265625, 0.50048828125, 1.0166015625, 0.99609375, -1.4921875, 0.442626953125, 0.7001953125, -1.01953125, -1.1376953125, 0.72265625, -1.908203125, 0.1080322265625, 1.908203125, -0.8525390625, 1.322265625, 0.06463623046875, 0.229248046875, -0.1488037109375, -0.256591796875, -0.37744140625, -0.5537109375, 0.74609375, 0.15869140625, 0.96826171875, 1.9228515625, 0.005756378173828125, -1.2421875, 0.96728515625, 1.205078125, 0.445068359375, 1.767578125, 1.1044921875, -0.17626953125, 2.05078125, 0.1593017578125, 0.27587890625, -1.158203125, -0.477294921875, 2.310546875, 0.96044921875, -0.65966796875, -0.294189453125, -0.2137451171875, 0.0723876953125, 0.72900390625, 1.4873046875, -0.63720703125, 1.162109375, -0.34326171875, 0.77001953125, 1.0126953125, 0.2054443359375, 0.5419921875, 0.90380859375, 0.1700439453125, -1.1650390625, -0.52734375, -0.0006885528564453125, -0.329345703125, 2.046875, -0.85302734375, -0.430419921875, 0.6943359375, 2.1328125, 0.209716796875, 0.8642578125, -0.1043701171875, -0.36669921875, 0.64501953125, 1.365234375, -1.4658203125, -1.09765625, 0.06591796875, -1.0419921875, 0.303466796875, 0.5869140625, 1.00390625, 0.37646484375, 0.1497802734375, 0.221923828125, 0.287353515625, -0.36474609375, -0.0966796875, -0.339599609375, 0.294189453125, -0.303955078125, 0.7919921875, 0.7001953125, -0.3955078125, -1.3642578125, 0.95068359375, -0.85693359375, -1.75, -1.220703125, 0.457275390625, -0.070556640625, -0.53955078125, -1.138671875, 0.1607666015625, -0.3046875, -0.332763671875, 1.189453125, -1.69921875, 1.6025390625, 0.328857421875, 1.2548828125, 0.5869140625, 0.85400390625, -0.677734375, 1.1611328125, -0.07659912109375, 0.82080078125, 0.8408203125, -0.595703125, 0.58984375, 0.172607421875, 1.4912109375, 0.39599609375, -0.31298828125, 1.34375, 0.5263671875, -0.16064453125, -2.736328125, -0.81396484375, 0.491455078125, 0.62548828125, -0.055908203125, -0.12939453125, 0.6640625, -0.8779296875, 0.0115814208984375, 0.96240234375, 0.314697265625, 0.025238037109375, -0.11785888671875, 0.8291015625, 0.95654296875, -0.490478515625, 2.5390625, -1.1962890625, -0.342529296875, 1.505859375, 0.376708984375, 0.033660888671875, 0.454345703125, -0.38525390625, 0.732421875, -0.388671875, 0.07049560546875, -0.72021484375, 0.84228515625, -0.3408203125, 0.712890625, -1.4482421875, -0.25732421875, -1.439453125, 1.185546875, -1.20703125, -0.845703125, 0.544921875, 1.0537109375, -1.9970703125, -1.4306640625, -0.56982421875, 0.30615234375, -0.046783447265625, 0.623046875, 0.1785888671875, 0.67626953125, 0.475341796875, 0.58447265625, -1.5087890625, 0.2705078125, -0.364501953125, 0.6845703125, -0.33740234375, -0.70849609375, -1.83203125, -0.80517578125, 0.39892578125, 0.78125, -0.7919921875, -0.16943359375, 0.91064453125, 0.6318359375, 1.462890625, -0.01413726806640625, 0.166259765625, 0.140869140625, 0.1070556640625, -0.494384765625, 1.212890625, -1.0908203125, -0.12042236328125, -0.1485595703125, 0.9453125, 2.84765625, -1.708984375, -0.3974609375, 0.07843017578125, 0.12371826171875, 0.2352294921875, -0.11895751953125, -1.0419921875, -0.37109375, -0.603515625, -0.17822265625, -1.6669921875, -0.03271484375, -0.642578125, 2.5390625, -1.259765625, 1.451171875, 0.7529296875, 0.0631103515625, 0.16748046875, 0.90625, -1.0234375, 0.1435546875, -0.40087890625, -0.75830078125, -1.5712890625, 1.83984375, -0.90087890625, 0.02386474609375, -0.8564453125, 1.626953125, 0.20068359375, 0.76318359375, -0.3701171875, 0.5390625, 0.3935546875, -0.1790771484375, -0.7626953125, 0.37890625, 0.10211181640625, -0.1915283203125, 1.353515625, -0.52734375, 0.71240234375, -0.400146484375, 1.935546875, 1.2890625, -0.490966796875, -0.32177734375, 0.2200927734375, 0.10052490234375, 2.966796875, -0.477294921875, 1.0556640625, -0.43408203125, -0.20458984375, 0.92822265625, -0.60205078125, -0.84375, 0.36328125, 0.6767578125, 1.177734375, 0.849609375, 0.6220703125, 0.72265625, 1.513671875, -0.94384765625, 0.9638671875, 3.173828125, 0.94140625, -0.97021484375, 0.37548828125, 2.12890625, -0.314697265625, 0.57177734375, -0.83056640625, -1.68359375, 0.6650390625, 1.1640625, -1.0, 1.068359375, -0.392822265625, 1.5693359375, -0.46728515625, -0.896484375, 0.1976318359375, 1.0302734375, 0.06195068359375, 0.398681640625, -1.7001953125, 2.748046875, -1.080078125, -0.92578125, -0.76416015625, 0.8056640625, 0.74658203125, -1.712890625, 0.327392578125, 2.0703125, -0.296630859375, -0.36279296875, 1.646484375, -1.4873046875, -0.83984375, -0.32763671875, -0.50244140625, -0.64697265625, -1.26953125, 2.65625, -1.578125, 0.037261962890625, 0.1370849609375, 1.1904296875, -0.916015625, 2.54296875, 1.0068359375, -0.6865234375, -1.1044921875, -0.66162109375, 0.96923828125, 1.810546875, 1.1923828125, -0.15576171875, -1.47265625, -1.75, -0.76025390625, 0.5361328125, -0.65087890625, 1.4599609375, -2.3203125, 0.2454833984375, -0.046356201171875, 0.1771240234375, 0.73046875, 1.2734375, -0.6611328125, 0.036590576171875, -0.29833984375, 0.80615234375, 0.9619140625, -0.65869140625, -1.3828125, 0.9111328125, 1.474609375, -1.6025390625, -0.57275390625, 1.4921875, 0.9521484375, -1.064453125, 0.00501251220703125, -0.0212249755859375, 0.1728515625, 0.049224853515625, 0.8154296875, 1.3701171875, -1.6865234375, 0.430908203125, -0.89111328125, -0.30712890625, 0.454833984375, -0.25390625, -1.150390625, 0.65625, -0.92236328125, 1.23828125, 0.35400390625, -0.392578125, -0.80859375, 0.376953125, 0.521484375, -1.1455078125, 0.454833984375, 0.61669921875, 1.48828125, 0.95458984375, 0.06494140625, 0.39404296875, -0.255615234375, 0.055816650390625, -0.77734375, -1.0322265625, 0.12103271484375, 0.94091796875, 0.1419677734375, -0.576171875, 0.83203125, -1.140625, 0.8330078125, 0.611328125, 0.12164306640625, 0.56591796875, -1.3955078125, 0.282470703125, 0.91259765625, 0.5126953125, -0.52783203125, 1.3251953125, -1.373046875, -0.8876953125, -0.8466796875, 0.7841796875, 0.73388671875, -0.241943359375, 0.5927734375, 0.607421875, -0.8271484375, 1.0087890625, -1.13671875, -1.84765625, -1.03515625, -0.5810546875, -0.69873046875, -1.6103515625, -0.8955078125, -0.1385498046875, 0.38134765625, -0.5947265625, -0.712890625, 0.6201171875, -0.1591796875, -0.043182373046875, -0.49560546875, -0.6298828125, 0.47021484375, -1.40234375, -0.488037109375, 1.0625, 0.375, 0.1947021484375, 0.58837890625, -1.251953125, -1.7802734375, 0.0836181640625, -1.7919921875, -1.857421875, -1.2529296875, 0.01160430908203125, -0.8515625, 0.5458984375, -0.2000732421875, -0.50537109375, -0.96435546875, -0.37548828125, 1.6884765625, -0.509765625, 0.12176513671875, -0.59130859375, -1.140625, 0.654296875, -1.58203125, 0.134765625, -1.26953125, -0.196044921875, 0.857421875, -0.0914306640625, -0.1636962890625, -0.1507568359375, -1.82421875, -0.73681640625, 0.342041015625, 0.67431640625, 1.7421875, -1.0615234375, 0.024169921875, 0.92724609375, -1.9755859375, -2.365234375, 0.1346435546875, 0.3017578125, 0.66943359375, -0.87646484375, 1.5693359375, -0.8369140625, -1.33984375, 0.1470947265625, -1.0146484375, -0.4208984375, -0.73974609375, 0.93212890625, -0.32666015625, -0.80908203125, 0.9921875, 0.68310546875, 0.869140625, 0.392822265625, -0.0263671875, 1.1044921875, -0.6748046875, -0.5966796875, -0.47802734375, 1.435546875, -0.7431640625, 1.76171875, -1.197265625, -1.9072265625, 0.50390625, 0.325927734375, -0.2042236328125, 1.3037109375, 1.1318359375, -0.1436767578125, -0.07666015625, -0.91162109375, 0.04901123046875, 0.321533203125, 1.40625, 0.9638671875, -0.04833984375, 0.0204620361328125, -1.2626953125, 0.87841796875, -0.337158203125, -0.1748046875, -0.634765625, -1.9609375, -0.212646484375, 0.165283203125, 0.75048828125, -0.5029296875, 0.6884765625, 1.064453125, 0.140380859375, -0.02313232421875, 0.1239013671875, -0.11328125, -1.21875, 0.568359375, 1.1904296875, 0.317626953125, 1.3828125, 0.04388427734375, 0.420654296875, -1.087890625, -0.60888671875, -0.400634765625, 0.4033203125, -0.9697265625, -0.360107421875, -2.296875, 0.337158203125, 1.01171875, 1.412109375, -0.0096893310546875, -0.5322265625, -0.459228515625, 0.25439453125, 1.87109375, 0.8671875, 0.5654296875, 1.7578125, 1.087890625, -0.7138671875, -0.25390625, 0.203369140625, -0.572265625, -1.310546875, -1.7607421875, -1.837890625, -0.56494140625, -0.018890380859375, 0.1248779296875, -0.414306640625, -1.462890625, -0.05755615234375, -1.287109375, 1.2099609375, 1.130859375, 0.427978515625, -1.1953125, -0.3896484375, 0.5302734375, -1.416015625, -1.7197265625, 0.05419921875, 1.8505859375, -2.02734375, -1.08203125, -0.9794921875, 0.142333984375, -0.03961181640625, 0.042694091796875, 0.5361328125, 1.4775390625, -0.4296875, -2.42578125, -0.64208984375, -0.6640625, -0.08380126953125, 0.462646484375, 0.73388671875, -0.50634765625, 1.8515625, -0.26318359375, -1.2548828125, -0.363525390625, -0.9833984375, -0.286865234375, 0.41259765625, 0.051116943359375, -1.7021484375, -0.60302734375, 0.56005859375, -0.626953125, -0.28515625, 0.006378173828125, 0.12042236328125, 0.87646484375, -0.525390625, 0.78466796875, 0.5576171875, 1.7314453125, -1.55859375, -0.216796875, 0.400634765625, -0.57958984375, -0.83740234375, 0.6298828125, -0.60498046875, 0.556640625, -0.288818359375, -1.462890625, -0.2064208984375, 1.2353515625, -0.099365234375, 0.525390625, -0.10577392578125, 1.1015625, 0.51611328125, -1.8798828125, 0.8486328125, 1.5888671875, 1.0625, -1.48046875, 0.0126495361328125, -0.52685546875, 0.56103515625, 0.0008363723754882812, 0.2322998046875, 0.438232421875, -1.3095703125, 1.3818359375, -0.396728515625, -0.409423828125, 0.640625, -1.3447265625, 0.85595703125, 0.3310546875, -1.064453125, 0.481201171875, -0.456787109375, -1.54296875, 0.1983642578125, 0.3486328125, 1.619140625, -1.0380859375, -1.51171875, 0.0138092041015625, -0.397705078125, -1.2548828125, -0.57080078125, -1.052734375, -0.90087890625, -0.082763671875, 0.5703125, 0.5078125, -1.04296875, 0.78857421875, 0.47021484375, -1.712890625, -1.0263671875, 0.1439208984375, 1.154296875, -1.19921875, 1.30078125, -0.0767822265625, 2.947265625, -1.1298828125, 1.10546875, -0.26318359375, -1.359375, 1.73046875, 1.0361328125, 0.258544921875, 0.64013671875, -0.439453125, -0.113037109375, 1.84375, -0.8515625, -0.544921875, 0.27099609375, 0.479248046875, -0.84814453125, -1.1064453125, 0.459716796875, -0.44140625, -2.0546875, -2.060546875, -0.26171875, -2.11328125, 1.2119140625, -0.50732421875, 0.6396484375, 0.0244903564453125, 0.28125, 0.20361328125, -0.67578125, 1.8876953125, -1.11328125, -0.051361083984375, 0.3173828125, 1.060546875, 0.28466796875, -0.2086181640625, 0.306884765625, 0.0018863677978515625, 0.489501953125, -0.88720703125, -0.97021484375, 2.0703125, -2.046875, -0.19873046875, 1.267578125, 0.60693359375, -0.623046875, 0.72607421875, -1.294921875, -0.054962158203125, 0.1859130859375, 2.447265625, 0.2406005859375, -0.1954345703125, 1.326171875, -1.251953125, -1.572265625, 1.7333984375, -0.66650390625, -0.77587890625, -1.2890625, 0.406982421875, -2.62890625, -0.62744140625, 0.82177734375, 2.076171875, 0.5810546875, 0.84619140625, -1.158203125, -2.353515625, 1.4404296875, 0.80322265625, -0.07037353515625, -0.771484375, 0.5068359375, -0.466552734375, 1.146484375, 2.2265625, 0.78173828125, -0.001739501953125, -1.2919921875, -1.04296875, 0.28173828125, 0.289794921875, -0.86083984375, -0.183837890625, 0.39697265625, -0.693359375, -0.49658203125, 0.396240234375, 1.0, -0.45751953125, -0.05657958984375, -0.66064453125, -0.4443359375, -0.2186279296875, 1.1064453125, -0.308837890625, -0.0821533203125, -0.052703857421875, 1.9384765625, -0.7412109375, 0.625, 0.092529296875, -0.423828125, 0.50732421875, 0.472900390625, 0.9755859375, -0.57080078125, 0.677734375, 0.6708984375, -0.34521484375, -1.525390625, -2.51171875, 0.418212890625, 0.11492919921875, 0.943359375, -1.4375, 1.39453125, 0.77001953125, -0.280517578125, -0.1937255859375, -0.22021484375, -1.3916015625, -1.0947265625, 1.716796875, 0.94384765625, -0.342041015625, 0.319091796875, 0.8837890625, 0.875, -0.297607421875, -1.33984375, 0.05889892578125, -0.2294921875, 1.0224609375, 0.1788330078125, -1.4658203125, -0.76806640625, 0.249755859375, -1.2353515625, -0.282958984375, -1.6181640625, 0.5478515625, -1.1298828125, -0.78857421875, 1.0078125, 0.884765625, 0.78369140625, -1.361328125, -0.1553955078125, -1.8349609375, -0.4375, 0.478271484375, 0.95947265625, 1.3369140625, 1.8544921875, 0.19580078125, -0.86962890625, -0.7021484375, 0.55712890625, 0.32958984375, 1.615234375, -0.7021484375, -1.5166015625, 0.337158203125, -0.7275390625, 0.56103515625, 1.7646484375, 0.2489013671875, 1.2099609375, -0.69921875, 1.3818359375, -0.57275390625, 0.6669921875, 0.054840087890625, 1.013671875, -0.1881103515625, 1.138671875, 0.304931640625, 0.52197265625, 0.304931640625, -0.88916015625, -1.5, 0.463134765625, -1.158203125, -1.4462890625, -0.61083984375, -1.447265625, 0.499755859375, -1.4697265625, 1.365234375, -1.5888671875, 0.65576171875, 0.333984375, -0.64990234375, 0.230224609375, 1.3271484375, -2.2109375, 0.8671875, -1.10546875, -0.041900634765625, 1.0947265625, -0.00765228271484375, -1.0751953125, 1.271484375, 0.2332763671875, 0.1971435546875, -0.78466796875, -1.2431640625, -2.015625, 1.6064453125, -1.185546875, 0.57421875, -0.3408203125, -0.09625244140625, -0.294677734375, 0.0516357421875, 0.164794921875, 0.10498046875, -0.77197265625, -1.8896484375, 0.2193603515625, -1.80859375, -0.40869140625, -1.44921875, 1.2998046875, 0.7431640625, 0.7294921875, -0.4462890625, -1.4267578125, 1.5625, -1.0615234375, 0.378662109375, 1.5361328125, 0.5966796875, 0.355224609375, 0.87158203125, -0.2398681640625, 0.174072265625, -0.52685546875, -0.11279296875, 0.7646484375, -2.609375, -0.1748046875, 0.87158203125, 1.044921875, 0.08404541015625, 0.251708984375, -0.053375244140625, 0.2073974609375, 1.822265625, -1.287109375, 1.484375, 2.306640625, -1.177734375, 0.9765625, 0.8642578125, 0.65087890625, 0.458251953125, 0.62158203125, -1.3916015625, 0.75390625, -1.251953125, 2.15234375, 0.84716796875, 0.78759765625, 0.101806640625, -0.56591796875, -0.492919921875, 2.7578125, 1.4697265625, 0.67724609375, -1.1865234375, -0.003742218017578125, 1.2802734375, -0.065673828125, 0.83544921875, -0.21337890625, 0.2203369140625, 0.6533203125, 0.2391357421875, -2.3046875, -0.373046875, -1.71875, 0.366455078125, -0.201904296875, -0.353271484375, -0.423583984375, -0.056304931640625, -0.453369140625, -0.01715087890625, 0.6279296875, 0.4970703125, -0.60888671875, -1.1201171875, -0.1026611328125, 0.65673828125, -3.828125, -0.35888671875, -1.25390625, -0.09259033203125, -0.9111328125, -0.9931640625, 0.1590576171875, 0.59033203125, -0.316162109375, 0.312255859375, -0.94091796875, -0.18017578125, -1.5654296875, -2.396484375, 1.7265625, -1.2509765625, -1.3154296875, -0.217041015625, -0.380126953125, -0.416748046875, 1.345703125, 0.548828125, -0.62939453125, 1.4140625, 0.4111328125, 2.39453125, -0.32421875, 0.407470703125, -0.6875, -0.2347412109375, -0.8662109375, -1.3994140625, -0.6064453125, 1.3916015625, 1.3046875, -0.1912841796875, -1.1484375, -0.126220703125, 0.236083984375, -0.468994140625, 0.455810546875, 0.283447265625, -0.84619140625, -0.69189453125, 0.38671875, -0.3388671875, 1.7431640625, -1.6904296875, 0.58251953125, 0.3212890625, 0.36376953125, 1.01953125, 0.2374267578125, 0.10418701171875, 0.205322265625, 0.12164306640625, -0.302978515625, -1.3037109375, -2.83203125, 0.0279693603515625, -0.78662109375, 0.496826171875, -1.404296875, 0.75439453125, -0.397705078125, 0.54248046875, 0.96728515625, 1.509765625, 0.379150390625, 0.70166015625, -1.3623046875, 1.671875, -0.73388671875, 0.58984375, -1.0166015625, 1.04296875, -0.2919921875, -1.140625, 0.7587890625, -0.79296875, -0.456298828125, 0.423583984375, 0.43701171875, 0.33544921875, 0.04705810546875, 0.7431640625, -1.806640625, -0.72265625, -1.23046875, -0.740234375, -0.399169921875, 0.78369140625, -0.094482421875, -1.451171875, -1.150390625, 1.33203125, 0.4443359375, -1.8173828125, -0.63134765625, 0.045135498046875, 0.350830078125, 0.1461181640625, 0.771484375, -1.775390625, 0.875, -0.48388671875, -1.0341796875, -0.49169921875, 1.0263671875, -0.2181396484375, -1.3408203125, -0.8017578125, -1.69140625, 1.2578125, 1.7333984375, -1.1572265625, 1.3076171875, 0.0012111663818359375, -1.30859375, 0.548828125, -1.60546875, 0.443603515625, 1.1904296875, -0.222412109375, -0.2401123046875, -0.334716796875, -0.0214996337890625, 1.5087890625, -0.1700439453125, 0.0172271728515625, -0.11260986328125, 1.2353515625, 0.87744140625, 0.2509765625, 0.68505859375, 1.6572265625, -0.82470703125, 0.2353515625, -0.47265625, 0.689453125, 0.207763671875, -0.152587890625, 1.240234375, 0.443603515625, -0.56396484375, -0.9453125, -0.29443359375, -0.89306640625, 1.4365234375, -1.27734375, 0.6484375, -0.8193359375, 1.56640625, -0.76806640625, 0.41650390625, 1.45703125, -0.8251953125, -0.32080078125, -0.58154296875, 0.0963134765625, -0.3271484375, 0.1898193359375, -0.97314453125, 0.3466796875, 0.47705078125, -1.224609375, 2.09375, 0.6474609375, 0.0621337890625, -2.30859375, -0.51220703125, 1.0615234375, 0.262939453125, -0.322998046875, -0.71826171875, -0.6220703125, -0.0380859375, -2.1015625, 0.283447265625, 0.298095703125, -0.4892578125, 0.2418212890625, -1.3583984375, 1.5263671875, -0.86376953125, -0.10595703125, 1.15625, 0.34228515625, -0.142822265625, 0.57763671875, 0.69482421875, -0.9248046875, -1.3447265625, 1.6875, -1.37890625, 0.33984375, 2.189453125, -1.3544921875, -0.38720703125, 0.60498046875, -0.9716796875, 0.2469482421875, -0.494140625, 1.134765625, -1.11328125, 0.72509765625, 1.1708984375, -1.142578125, 0.4130859375, 2.044921875, -2.3984375, -1.203125, -1.5634765625, -0.99560546875, -1.587890625, 1.337890625, -0.7021484375, 0.6689453125, -0.238525390625, -0.6953125, 0.82080078125, -0.0869140625, -0.6953125, -1.9189453125, 1.265625, -0.8779296875, 0.427734375, 0.189697265625, -0.270263671875, -0.84130859375, 2.076171875, -0.1746826171875, 0.47998046875, 1.0947265625, -0.24462890625, -0.86669921875, -1.0068359375, -1.875, -0.86865234375, -2.466796875, 0.8662109375, -0.84130859375, -0.7802734375, 0.88623046875, 1.5126953125, -0.87841796875, 0.1016845703125, -0.99462890625, -0.6640625, 0.08892822265625, -0.62646484375, 0.1822509765625, 1.0, 0.12298583984375, -1.4501953125, -0.7587890625, -0.264892578125, -1.1201171875, -0.7998046875, -0.449462890625, -0.66162109375, -2.259765625, -1.08984375, -1.244140625, 0.66748046875, -1.0224609375, 0.326416015625, 0.20458984375, -0.00274658203125, 1.13671875, -1.984375, 1.7666015625, 0.3583984375, 0.4443359375, 0.71728515625, -0.066650390625, -1.4873046875, 0.7724609375, -0.54833984375, -0.398193359375, -1.109375, -1.3251953125, -1.0234375, 0.1097412109375, -0.207763671875, -1.0703125, -1.2626953125, -0.65966796875, -0.7802734375, -0.99560546875, 0.6240234375, 0.7177734375, 1.728515625, 1.681640625, 0.35888671875, -0.1766357421875, -1.0546875, -0.3076171875, 0.1363525390625, 0.230712890625, 0.1419677734375, -0.1402587890625, 1.7265625, 0.53564453125, -0.0653076171875, -0.73388671875, -1.1728515625, 2.359375, 0.99853515625, -1.09765625, 0.448974609375, 0.2646484375, 0.9638671875, 0.349853515625, -0.1617431640625, -0.156982421875, 1.154296875, -0.36767578125, 0.931640625, -0.06927490234375, 0.560546875, -1.392578125, -0.035858154296875, -0.43896484375, -1.2841796875, -0.65283203125, 0.27392578125, -0.87158203125, -1.138671875, -0.41015625, 0.14599609375, 0.146484375, 1.6474609375, 0.67333984375, 1.6005859375, -1.201171875, 0.455322265625, -1.93359375, -0.66455078125, 0.3154296875, 1.48828125, -0.6845703125, 0.122314453125, 0.37744140625, 0.1607666015625, -0.337646484375, 0.50537109375, 1.5068359375, 0.1832275390625, -2.0234375, -1.0537109375, -0.21044921875, -1.6025390625, -2.3359375, 0.9404296875, -0.92724609375, -1.2216796875, 0.6259765625, -0.94189453125, -0.11981201171875, -0.53125, 0.69384765625, -0.291015625, 0.6171875, -0.150146484375, 0.75244140625, -1.853515625, 0.356689453125, -1.18359375, 0.394775390625, -0.50048828125, -0.8408203125, 1.0244140625, -0.29052734375, 0.845703125, -1.5859375, 0.381103515625, 2.06640625, -1.046875, 0.7333984375, -1.1611328125, 0.2274169921875, 0.60205078125, -1.25, 1.6748046875, -0.1439208984375, -0.353759765625, -1.5029296875, -0.56884765625, -0.441650390625, 0.6494140625, -1.8310546875, 0.55322265625, 0.1466064453125, -1.1162109375, 1.513671875, 0.09283447265625, 1.9765625, 1.328125, -0.32470703125, 0.69482421875, 1.1552734375, 1.4345703125, -0.1236572265625, 1.0888671875, 0.6162109375, -1.9990234375, -0.53564453125, 0.98193359375, -0.5048828125, -0.99462890625, -1.2685546875, 0.56494140625, -1.4052734375, -0.112548828125, 0.53125, 0.01067352294921875, -1.048828125, 1.21484375, -0.296142578125, 0.12060546875, 1.265625, 1.0888671875, 1.5966796875, -1.5595703125, -0.143310546875, -0.37255859375, 0.8359375, -0.28369140625, 1.5068359375, -1.0517578125, -0.1673583984375, 1.1640625, -0.02178955078125, 0.2000732421875, 0.255126953125, -0.4716796875, -0.233642578125, 1.4365234375, 1.2822265625, -0.263671875, 0.39990234375, -0.2919921875, -0.451416015625, 1.2529296875, 0.18798828125, -1.4052734375, -0.54541015625, -1.4931640625, 0.56201171875, -0.5830078125, -0.45703125, 1.443359375, 1.794921875, -0.83203125, -1.8486328125, 0.9853515625, -0.1702880859375, -0.1080322265625, -0.8994140625, -0.241455078125, -3.029296875, 1.1640625, 1.962890625, 0.54443359375, 1.0966796875, 0.40478515625, -1.7294921875, -0.34765625, 0.59423828125, 1.27734375, -0.279541015625, -0.14404296875, -0.52001953125, -0.60302734375, 0.73046875, 0.89111328125, 2.314453125, 1.16015625, -0.63330078125, 0.08526611328125, 1.5478515625, 0.206787109375, -0.316650390625, 1.5302734375, 1.216796875, 1.4130859375, -2.470703125, 0.286865234375, -0.7578125, 1.5888671875, -0.955078125, 1.0927734375, -0.45947265625, -0.1641845703125, 0.00899505615234375, 1.1611328125, 0.98681640625, 0.75, -1.0498046875, 0.60107421875, -0.75048828125, -0.1395263671875, 1.4580078125, 0.99755859375, -0.58544921875, -1.1015625, 0.705078125, -0.751953125, 0.56689453125, -0.205078125, -1.8857421875, -0.8046875, 0.1993408203125, -1.1337890625, 0.059661865234375, 0.44287109375, 0.1646728515625, 2.03125, 0.25390625, 0.4921875, 0.8994140625, -2.0, 0.06787109375, 0.438232421875, 0.45458984375, 0.282470703125, 0.55322265625, 1.0859375, 0.8720703125, -0.328369140625, 0.75830078125, -0.415771484375, -0.3837890625, -0.0011138916015625, -0.49365234375, -1.62109375, 0.55029296875, 0.84814453125, -1.162109375, -0.268798828125, 2.560546875, 0.6279296875, 0.141357421875, 0.560546875, 0.047637939453125, -2.10546875, -0.2320556640625, 0.494140625, 0.1043701171875, 0.93017578125, 1.0146484375, 1.341796875, -0.0303497314453125, -0.89892578125, 1.962890625, 1.35546875, -0.405029296875, 1.513671875, 2.94140625, -1.8125, 0.1546630859375, -0.044525146484375, 0.042388916015625, 0.87353515625, 0.98291015625, 0.005107879638671875, -0.5048828125, -0.230712890625, 0.87890625, -1.1533203125, 1.5986328125, -1.6962890625, -2.41015625, 0.273681640625, 0.116455078125, -0.1278076171875, -2.0, 1.1015625, -0.87158203125, 0.3134765625, 0.03021240234375, 2.1171875, -0.67236328125, 1.3212890625, 0.1356201171875, 1.3408203125, 0.372802734375, -1.4892578125, -0.360107421875, 0.46142578125, 0.436767578125, 0.13232421875, -0.71728515625, 0.541015625, -2.103515625, -0.462646484375, 1.583984375, -0.9716796875, 0.1287841796875, 1.484375, 0.65869140625, -1.3046875, 0.6513671875, 1.208984375, -0.89697265625, -0.98388671875, -1.576171875, 1.38671875, 1.4814453125, -0.67822265625, -0.98095703125, -0.0745849609375, 0.8896484375, 0.623046875, -1.02734375, 1.6669921875, 0.90576171875, 0.52685546875, 1.001953125, -1.3515625, 1.822265625, -1.259765625, -0.51904296875, -0.6240234375, 1.5625, -2.626953125, 0.681640625, -2.029296875, -0.329345703125, 0.6611328125, -0.28955078125, -1.3134765625, 0.62841796875, -1.521484375, -0.1279296875, -0.66943359375, -2.09765625, -0.1551513671875, 0.045684814453125, -1.919921875, -0.166748046875, 2.0625, 0.6103515625, -0.07122802734375, -0.96923828125, -0.54736328125, -0.453369140625, 0.76171875, 1.1669921875, 0.1199951171875, 0.54150390625, 1.0048828125, -0.74658203125, 0.6142578125, -1.0146484375, -0.230224609375, -0.497802734375, 1.4228515625, 0.218994140625, -0.81982421875, 0.47705078125, -0.65673828125, -1.7314453125, -0.467041015625, -0.387939453125, 1.541015625, -1.0908203125, 0.0906982421875, -1.0673828125, 0.2247314453125, 0.2271728515625, 0.168212890625, 1.33203125, -1.6259765625, 2.75, -0.7666015625, 0.06610107421875, 0.1524658203125, 1.3798828125, -0.5537109375, 2.3515625, 0.0740966796875, -0.06158447265625, 0.412353515625, 0.62939453125, 0.186767578125, -0.1368408203125, 0.5947265625, -0.151123046875, 1.07421875, 2.6640625, 1.296875, -1.220703125, 1.1748046875, -0.62646484375, 2.17578125, 1.1484375, 0.196533203125, 0.740234375, -0.5322265625, -0.29931640625, -1.5361328125, -1.0908203125, -0.70703125, -1.80078125, -0.138671875, 1.4609375, -0.2159423828125, -0.2440185546875, 0.5146484375, 0.95166015625, -1.076171875, 0.404541015625, 0.0797119140625, -0.578125, 0.7294921875, 0.82763671875, 1.0615234375, -0.56494140625, 0.377197265625, 0.8330078125, 0.97607421875, -1.3046875, -2.166015625, 0.625, 0.9228515625, 0.5986328125, -0.40625, 0.62060546875, 0.076904296875, 1.294921875, -1.7939453125, -1.794921875, -0.544921875, -0.051544189453125, 0.1397705078125, -2.19921875, 0.327392578125, 1.9072265625, 0.63818359375, -0.049407958984375, 0.69921875, 1.3212890625, -0.92919921875, -1.15625, -0.7900390625, -0.0022983551025390625, -0.1279296875, -1.048828125, -2.529296875, 0.2041015625, 0.305419921875, 0.86767578125, -1.841796875, 1.048828125, 1.125, 0.44873046875, 0.79541015625, -0.321044921875, -1.3603515625, -2.43359375, -1.5546875, 0.53857421875, 0.1417236328125, 0.599609375, 1.1123046875, 1.228515625, -0.52099609375, -0.435302734375, -0.06268310546875, 0.75927734375, 1.021484375, 1.1533203125, 0.12646484375, 0.7060546875, -0.86181640625, 2.033203125, -0.52685546875, 0.9228515625, 0.10833740234375, -0.0251312255859375, -0.036407470703125, -1.0751953125, -0.30419921875, -0.357421875, -0.203857421875, -2.216796875, 1.1865234375, -1.080078125, -1.0244140625, -0.374755859375, -1.6884765625, 0.177001953125, -0.003101348876953125, 0.50634765625, 0.09918212890625, -0.61669921875, -0.6455078125, -0.440673828125, 0.37158203125, 1.015625, 1.513671875, -0.69677734375, -0.73974609375, 1.009765625, 0.498779296875, -0.2587890625, 0.136474609375, -0.1322021484375, 0.5888671875, 0.1512451171875, 0.307861328125, -0.219482421875, 0.65625, -1.9716796875, -1.37890625, 0.640625, -0.59716796875, -2.017578125, 0.184814453125, -0.6640625, -0.63037109375, -0.1441650390625, -0.07122802734375, 0.87890625, -1.970703125, -1.396484375, 1.7158203125, 0.05096435546875, -0.72705078125, -1.609375, 2.025390625, 0.00771331787109375, -0.5927734375, -0.07806396484375, 0.6044921875, -0.13671875, -0.9345703125, 1.0576171875, 1.3544921875, 0.50244140625, 0.7275390625, -0.7578125, 0.1458740234375, 0.50048828125, 0.002655029296875, 0.1605224609375, 0.6513671875, -0.11053466796875, 1.0966796875, 0.52490234375, -1.078125, -0.040008544921875, -1.0283203125, 0.8125, -0.81591796875, 1.4609375, 2.28125, 0.030609130859375, 0.377685546875, 0.931640625, 0.53759765625, -1.1259765625, -0.30322265625, -2.498046875, -0.171875, 0.1324462890625, 0.294189453125, 0.2005615234375, -0.6611328125, 2.6328125, 1.16796875, -0.30859375, 0.351806640625, -0.4833984375, -0.48681640625, 0.06744384765625, -0.0577392578125, 0.255859375, -0.19091796875, -0.06298828125, 0.444091796875, -0.205078125, 1.50390625, -1.033203125, 1.03125, 1.419921875, 0.74658203125, 0.226318359375, -0.2369384765625, 0.712890625, 0.80029296875, 1.125, -0.8349609375, -0.1895751953125, 0.81787109375, -0.6962890625, -0.14794921875, 0.96337890625, 0.681640625, -0.2459716796875, -0.1163330078125, -0.994140625, 1.8466796875, 0.59619140625, -1.1220703125, -1.755859375, -0.82177734375, 0.8271484375, -2.759765625, -1.080078125, -2.318359375, 0.10162353515625, -0.54833984375, 0.1622314453125, 0.277587890625, -1.8916015625, 0.97216796875, -0.2705078125, 0.69970703125, 0.85693359375, -1.158203125, 1.8466796875, -0.767578125, -0.4697265625, -0.57275390625, 0.7158203125, 0.2548828125, 1.26953125, -1.052734375, 0.269775390625, -0.2919921875, -0.96240234375, -2.166015625, -1.2685546875, 2.388671875, 1.7646484375, 0.28466796875, 0.333251953125, 1.4111328125, -0.01384735107421875, -0.135009765625, 0.2327880859375, -1.74609375, -2.671875, 0.06494140625, -0.174560546875, 2.494140625, -0.258544921875, -0.62353515625, -0.497314453125, -1.37109375, 1.26953125, -1.9658203125, 0.8857421875, 0.62841796875, -0.34814453125, -0.390380859375, 0.11676025390625, -0.63427734375, 0.28857421875, 0.564453125, -0.23486328125, 1.2548828125, 1.015625, -0.296875, 0.32373046875, -0.82177734375, 0.1529541015625, 0.640625, -0.1964111328125, 0.337890625, 1.337890625, 0.470458984375, 0.71142578125, 1.1240234375, 1.0263671875, -0.58203125, -0.51708984375, -0.335693359375, 0.0845947265625, 0.2734375, -1.0380859375, 0.5087890625, 0.36376953125, 0.81787109375, -1.00390625, -0.9345703125, 1.0224609375, 1.4384765625, 0.173828125, -0.89013671875, 0.7529296875, -0.3701171875, -1.39453125, -1.236328125, 0.681640625, -1.0966796875, -1.873046875, -1.4794921875, -0.63525390625, -0.732421875, -1.033203125, -1.9228515625, -0.058685302734375, 1.16015625, -0.81201171875, -0.36962890625, -0.5302734375, -0.12164306640625, -1.576171875, -0.292724609375, 0.1859130859375, -0.74169921875, -0.424560546875, 0.73388671875, -0.307861328125, -0.07818603515625, 0.0323486328125, 0.38427734375, -0.308837890625, -0.927734375, -1.748046875, 0.90478515625, 1.3525390625, 1.9345703125, -1.095703125, 0.8203125, -0.833984375, 0.27685546875, 0.646484375, 0.11114501953125, 0.93505859375, 0.5400390625, -0.2381591796875, 0.51611328125, -0.9248046875, -0.06768798828125, 0.4365234375, 0.444580078125, -2.193359375, -0.274658203125, -0.64306640625, -0.27099609375, 0.56005859375, 0.414794921875, -0.84765625, 1.466796875, -1.4296875, -1.3408203125, 0.273193359375, -0.3828125, -2.08984375, 2.3359375, -0.71142578125, 0.482421875, 2.05859375, 2.779296875, -0.5478515625, -0.36474609375, -1.2294921875, 0.935546875, 0.728515625, 0.92626953125, -0.209228515625, -0.529296875, -1.0263671875, 1.7177734375, -1.625, 1.466796875, 1.0205078125, 0.052154541015625, -0.055145263671875, 1.353515625, -1.0234375, 0.255615234375, -0.630859375, -1.8037109375, -1.623046875, -0.58056640625, 0.81884765625, 0.93115234375, -1.1640625, 0.537109375, 0.59716796875, 0.0215301513671875, 2.12109375, 0.12310791015625, -1.068359375, -0.169189453125, -0.273193359375, 1.0390625, 0.2861328125, 0.348388671875, -0.85693359375, -1.37109375, -0.45703125, 1.0009765625, 1.177734375, -0.067626953125, 0.211669921875, -1.9697265625, 0.45654296875, -1.2763671875, -0.69873046875, -0.99462890625, -0.8388671875, -0.80517578125, -0.6572265625, 0.572265625, -1.240234375, 1.0263671875, 1.23828125, 0.98583984375, 0.49365234375, 0.00473785400390625, 0.30419921875, 0.755859375, -0.74853515625, -0.791015625, 0.599609375, -0.17724609375, -2.091796875, 0.505859375, 1.611328125, 0.7626953125, -1.4130859375, -0.75048828125, 0.385009765625, -0.0013036727905273438, 1.427734375, -0.8212890625, 0.25390625, -0.9833984375, -1.2431640625, -1.931640625, 1.1650390625, 0.23681640625, -0.131591796875, -0.44482421875, -1.3583984375, 0.12274169921875, -0.2471923828125, 0.3447265625, -0.62841796875, 2.126953125, -0.51123046875, -1.0927734375, -0.305908203125, 0.76123046875, 1.498046875, -1.0234375, -1.416015625, -1.0771484375, -0.0911865234375, 0.36181640625, 0.83251953125, 0.456298828125, 0.02410888671875, -0.1368408203125, 0.00641632080078125, -0.896484375, 0.0083160400390625, -1.1328125, -0.53271484375, 0.6943359375, -1.041015625, 0.9599609375, -0.70166015625, -1.3701171875, -1.4619140625, 0.5185546875, -1.2119140625, 1.2666015625, 0.53564453125, -1.2666015625, -0.80810546875, 1.87890625, 0.42919921875, 0.40576171875, -0.5029296875, -1.1748046875, -1.5810546875, -1.3544921875, 1.427734375, 0.081298828125, -1.8837890625, -2.337890625, -0.219482421875, -0.11224365234375, 1.880859375, 2.0546875, 0.0728759765625, 1.546875, 1.2177734375, 0.12353515625, -0.02581787109375, 0.89404296875, 0.255615234375, 0.44775390625, -1.3603515625, -1.4296875, -0.2119140625, -0.07037353515625, -0.64013671875, 0.701171875, 1.1337890625, -0.6767578125, 0.29345703125, 0.331787109375, -0.18115234375, -2.4140625, -1.1240234375, 1.5703125, 0.0192108154296875, 0.409912109375, 1.4150390625, 0.0310821533203125, 1.130859375, 1.376953125, -0.01067352294921875, -0.7568359375, -1.7294921875, -0.22021484375, 1.6171875, 0.44140625, -0.9052734375, 0.30859375, -0.81591796875, -1.0068359375, 0.55126953125, 1.8857421875, 0.52783203125, -0.326904296875, -1.6923828125, 0.07763671875, -0.328125, 0.382568359375, -0.98095703125, -1.361328125, 0.35986328125, -1.0322265625, 1.060546875, 0.0196990966796875, 0.2069091796875, -0.3193359375, 0.480224609375, 2.04296875, -1.0205078125, 0.305419921875, -0.9521484375, -0.89697265625, 0.08465576171875, -1.5126953125, -0.0106964111328125, -0.50732421875, 0.72412109375, -0.7392578125, -1.46484375, -0.75048828125, 0.88427734375, -1.3134765625, -0.46337890625, -2.2734375, -0.798828125, -0.53076171875, 0.4638671875, -0.595703125, -1.5634765625, 0.50634765625, -0.89599609375, -0.0311279296875, -0.26953125, -0.275634765625, 0.974609375, 0.81982421875, 0.34716796875, -0.8759765625, 0.501953125, -0.43994140625, 1.1650390625, 1.541015625, -1.6962890625, 0.96923828125, 0.1590576171875, 1.3037109375, 0.9931640625, -0.442138671875, 0.06890869140625, -0.452880859375, -0.5556640625, -0.26953125, 0.78759765625, 1.3984375, 0.486328125, -1.6884765625, -0.330078125, 0.08392333984375, -0.0160064697265625, -0.39501953125, 1.1162109375, 1.7958984375, -0.64990234375, 0.93212890625, 0.2386474609375, -0.3994140625, 0.89599609375, 0.6591796875, -0.7646484375, 1.4677734375, 0.301025390625, 0.327392578125, -0.312744140625, -0.24267578125, -1.5947265625, 0.475830078125, -0.31689453125, 0.66455078125, 0.0836181640625, -0.2193603515625, -0.80322265625, -0.0013856887817382812, -0.72900390625, -1.3212890625, -0.320556640625, 0.52294921875, 0.1932373046875, 0.14453125, -0.468017578125, 0.94580078125, 0.0015249252319335938, 1.076171875, 1.1669921875, -0.4033203125, 0.18017578125, -0.73291015625, 0.495849609375, 0.105224609375, 0.42041015625, 1.0234375, 1.689453125, 0.05596923828125, -0.479736328125, 0.77490234375, -0.517578125, 0.268310546875, -0.453125, -0.54443359375, -1.4296875, 2.255859375, -0.51513671875, -1.4228515625, -0.115966796875, -2.20703125, 0.7900390625, -1.5009765625, 0.54443359375, -0.98095703125, 1.2841796875, 0.92919921875, -1.1005859375, -1.59375, 0.6328125, -1.123046875, -1.1826171875, 1.275390625, -0.4248046875, 0.048614501953125, 0.34716796875, -1.1533203125, -0.1573486328125, -2.458984375, 0.9482421875, -1.6572265625, -0.36279296875, -0.1175537109375, 0.69580078125, -1.0283203125, -1.1220703125, 2.216796875, -0.70458984375, -1.2421875, -0.039154052734375, 1.529296875, 0.438232421875, -1.3701171875, -0.1617431640625, 0.1883544921875, -0.94189453125, -0.304931640625, -0.73779296875, 0.5703125, -1.0439453125, -1.181640625, 0.91357421875, 0.424072265625, 1.90234375, 0.197265625, 0.356201171875, 0.40087890625, -0.83837890625, 0.94091796875, 1.359375, -0.71875, -2.1171875, -0.2120361328125, 2.16796875, 1.6572265625, 1.30859375, 0.3330078125, -0.93701171875, -0.72021484375, 2.642578125, 2.580078125, 0.275146484375, 0.82470703125, 0.058441162109375, 0.10479736328125, 0.673828125, -1.11328125, 0.303955078125, -1.052734375, 0.90234375, 0.46630859375, 0.353515625, -0.355224609375, -0.70849609375, -2.361328125, 0.35498046875, -0.34912109375, -0.927734375, -1.0859375, -1.5830078125, -1.748046875, 1.1015625, -0.199462890625, 0.01247406005859375, -0.37646484375, 0.6552734375, -0.1656494140625, 0.429443359375, 0.0848388671875, 1.3779296875, 0.353271484375, -0.67919921875, 0.98486328125, -0.529296875, -1.3046875, -1.6767578125, -1.775390625, 0.380126953125, -1.1650390625, 1.009765625, 0.237548828125, -1.142578125, -2.525390625, -1.2275390625, 1.099609375, -0.125732421875, 0.09222412109375, 1.08203125, -0.75732421875, 0.2049560546875, -0.7158203125, -0.427978515625, 0.1658935546875, -0.4990234375, -1.4296875, 0.74267578125, 1.2314453125, -0.005237579345703125, -2.298828125, 0.6083984375, -0.299560546875, 0.2042236328125, 0.69189453125, -0.11041259765625, -0.28662109375, -2.478515625, -1.24609375, -0.6083984375, -0.50634765625, 2.14453125, 0.921875, 2.345703125, -1.021484375, 0.00756072998046875, 0.89013671875, -0.2939453125, 1.7275390625, -0.1915283203125, -2.044921875, 0.002140045166015625, -1.5888671875, -0.405517578125, -0.2254638671875, 0.580078125, -2.08984375, 0.71240234375, -0.73388671875, -1.8671875, 1.298828125, 0.8193359375, -0.69677734375, 0.99609375, -0.1961669921875, 0.98681640625, -0.626953125, 1.2509765625, -2.10546875, -0.95263671875, -0.6513671875, -0.86767578125, -0.2095947265625, 0.16455078125, 0.73779296875, 0.432861328125, -1.4345703125, -0.0005483627319335938, 1.2080078125, 0.03240966796875, 0.060394287109375, 0.461669921875, -0.18310546875, -0.470947265625, -1.6123046875, 0.9404296875, 1.3564453125, -2.44140625, -0.50439453125, -0.50244140625, -0.394775390625, 0.138916015625, -1.181640625, -1.2177734375, -0.28955078125, 1.1787109375, 1.4326171875, 0.771484375, -2.138671875, 0.802734375, 0.6455078125, -0.0933837890625, -1.2451171875, 1.75390625, 0.7197265625, 0.98388671875, -0.134521484375, -0.334228515625, -0.1983642578125, 0.344482421875, -0.6005859375, 0.7724609375, -0.51904296875, -0.568359375, 1.1484375, 0.4384765625, 0.564453125, -0.7861328125, -1.349609375, -1.1669921875, 0.9580078125, -0.330322265625, 1.8427734375, 0.306396484375, -0.17919921875, -2.052734375, 0.053070068359375, -0.7734375, 0.9814453125, 1.392578125, -0.7216796875, -0.50048828125, 1.6240234375, -2.03125, -0.026519775390625, -0.1588134765625, -0.72705078125, 1.40234375, -0.1566162109375, -1.3427734375, -0.2027587890625, -0.6220703125, -0.63525390625, -0.7666015625, 1.1357421875, -0.400390625, 0.319091796875, 1.5791015625, 0.1566162109375, -2.890625, 0.8046875, 0.86474609375, 0.31396484375, -0.505859375, -0.0943603515625, -2.900390625, -1.048828125, -1.1328125, -0.463134765625, -0.431396484375, -1.7646484375, -1.171875, 2.224609375, -1.595703125, -1.0244140625, 0.17138671875, 0.9892578125, -0.409423828125, 0.54248046875, -0.1800537109375, 1.490234375, -1.3759765625, -2.39453125, 0.10791015625, 0.390869140625, 0.68408203125, -0.5576171875, 0.32373046875, 0.87939453125, -0.360107421875, 0.87060546875, -0.07989501953125, -0.669921875, -0.2437744140625, -0.6455078125, -0.80078125, 0.572265625, -0.41845703125, 1.029296875, 0.421142578125, 0.57568359375, -1.228515625, 0.35400390625, -1.220703125, -0.93359375, -1.7255859375, -0.440673828125, 0.046051025390625, -0.619140625, 1.1376953125, 0.268798828125, -0.27734375, 0.8828125, -0.44970703125, -0.487548828125, 0.85693359375, -0.289306640625, 0.57177734375, -0.09429931640625, -1.8115234375, -1.5244140625, 1.0634765625, -0.403076171875, 0.5361328125, 1.9326171875, -3.017578125, -1.3515625, -1.9443359375, -0.1881103515625, 0.42236328125, 0.9619140625, 0.0196075439453125, -0.91162109375, -1.2900390625, 0.01474761962890625, -1.3837890625, -0.1435546875, 0.3330078125, -1.7333984375, -2.17578125, 1.146484375, 0.4208984375, 0.262939453125, 1.0166015625, -2.541015625, -0.363525390625, -1.966796875, -1.24609375, -1.501953125, 0.456298828125, -0.01435089111328125, -0.6904296875, -1.5927734375, 0.425537109375, -1.279296875, -0.218505859375, -0.216064453125, -0.38134765625, 0.7216796875, 0.25048828125, -0.2490234375, 0.5546875, 0.1827392578125, 0.11480712890625, 0.5478515625, 0.236083984375, 1.12109375, -0.01012420654296875, -0.833984375, -0.2127685546875, -0.8251953125, 0.4619140625, -0.274658203125, 1.857421875, -0.0022449493408203125, -0.55908203125, -0.6962890625, 0.343017578125, -1.44921875, -0.52880859375, -0.029632568359375, 0.382080078125, 0.239990234375, 0.8466796875, 0.5283203125, -0.5078125, -0.51123046875, 0.355224609375, -0.285400390625, -1.7548828125, 0.90966796875, 0.178955078125, -1.26953125, -0.63671875, 0.30029296875, -0.0670166015625, 0.44482421875, 1.49609375, 1.271484375, -0.34765625, 1.4111328125, 0.460693359375, 0.65234375, 0.0099639892578125, -0.0726318359375, 0.7041015625, -0.454833984375, 1.037109375, -1.09765625, -0.06683349609375, 0.218505859375, 0.401123046875, 0.88916015625, -0.161376953125, 1.904296875, 0.14208984375, 0.2340087890625, -0.6689453125, -0.9990234375, 0.91552734375, -1.08203125, 0.533203125, -1.7919921875, 1.474609375, -1.2509765625, -0.115478515625, 1.09375, 0.833984375, 0.82470703125, 0.28564453125, -0.18212890625, -0.9345703125, 0.56298828125, 1.302734375, -0.2225341796875, -0.96044921875, -0.51806640625, 0.457275390625, 0.56298828125, 1.4140625, 0.9931640625, 0.2200927734375, -1.509765625, 0.91064453125, -1.9423828125, -0.374267578125, -1.083984375, -0.312255859375, -0.0672607421875, 1.517578125, 1.029296875, 0.6103515625, 1.5234375, 0.078125, 0.039093017578125, -0.35791015625, -0.55712890625, -0.97802734375, 0.1434326171875, 1.1025390625, 1.0029296875, 0.03192138671875, 1.1494140625, 1.1318359375, -0.164794921875, -1.072265625, 0.67919921875, 0.3740234375, 0.304443359375, -0.1993408203125, -2.09765625, -0.828125, 0.1702880859375, 0.212646484375, 1.0185546875, -0.96484375, 1.3984375, -1.2333984375, 2.009765625, -0.279296875, 0.5810546875, -0.32470703125, 0.970703125, -2.134765625, -0.428466796875, -1.09375, -0.625, -0.479736328125, 0.802734375, 1.248046875, 0.85546875, 1.5888671875, 1.0029296875, -1.09375, -0.5498046875, 0.56884765625, 1.7724609375, 1.0126953125, 2.201171875, 0.90478515625, 2.015625, 0.0078582763671875, -0.7060546875, 0.2095947265625, -0.04888916015625, 0.2685546875, -0.5771484375, -0.441162109375, -0.85009765625, -1.12890625, 1.4970703125, -0.255615234375, 2.81640625, -1.9736328125, 0.5498046875, -0.359375, 1.896484375, 0.2020263671875, 0.229736328125, -0.76953125, 1.0205078125, -0.56005859375, -1.9189453125, 0.13671875, 0.68505859375, 0.470703125, 0.80712890625, 0.73974609375, -0.3876953125, -0.412109375, 1.748046875, 1.9140625, 1.30859375, 0.8642578125, -0.5625, -0.1956787109375, -0.293701171875, 0.50048828125, -0.484375, -1.7578125, 1.7939453125, 1.1640625, -0.572265625, -0.3173828125, -0.6513671875, -0.5283203125, 0.1885986328125, -1.1611328125, 2.255859375, -0.95654296875, 0.2042236328125, 1.03515625, -0.6123046875, -1.666015625, -0.7958984375, -0.289794921875, -0.1871337890625, -0.90234375, 0.0947265625, 0.43408203125, 0.451171875, -0.185791015625, -0.273681640625, 0.93896484375, 0.39453125, 1.9404296875, -0.37744140625, 0.666015625, -0.24560546875, -0.01171875, -0.85791015625, 1.1162109375, -1.5166015625, 0.46044921875, 0.7109375, -0.79541015625, 2.845703125, 1.7822265625, -0.517578125, -2.505859375, 1.263671875, 1.0859375, 1.7978515625, -1.2587890625, 0.058990478515625, -0.37109375, -1.2392578125, -0.68115234375, -0.364990234375, 1.5615234375, -0.84521484375, -0.150390625, -1.4921875, 2.37109375, 0.433349609375, -0.3076171875, 0.60791015625, 0.92578125, 1.419921875, -0.95068359375, -0.57275390625, 0.9453125, -1.4921875, -1.1552734375, -1.58203125, 0.982421875, -1.0859375, 0.45947265625, 1.025390625, -2.330078125, 1.4599609375, 0.65625, -0.062225341796875, -1.1123046875, -0.583984375, 0.145751953125, 0.047149658203125, -0.5078125, 0.2744140625, 0.814453125, -0.7392578125, -0.373291015625, 0.474365234375, 0.0687255859375, 0.046051025390625, 0.9189453125, 0.39794921875, -1.1728515625, 0.84375, 1.9658203125, -0.2188720703125, -0.1246337890625, -0.83935546875, 0.99658203125, 1.1708984375, -1.12890625, 0.0120086669921875, 0.190185546875, -0.57421875, -1.5712890625, -2.0390625, 0.246826171875, -0.52978515625, -0.59814453125, -0.15283203125, 0.66259765625, 0.5263671875, 0.40771484375, 0.08612060546875, 0.354736328125, -0.385009765625, -1.1123046875, 2.455078125, 0.38720703125, 0.376708984375, 0.1834716796875, 1.1240234375, 0.226318359375, -1.8212890625, 0.428955078125, -0.2081298828125, -0.56689453125, -1.0166015625, 0.69091796875, 0.255126953125, 0.08465576171875, 0.2218017578125, 0.2410888671875, -0.16357421875, 1.0244140625, 0.190185546875, 0.298095703125, -0.499267578125, 0.81982421875, 2.041015625, -1.2373046875, -0.56982421875, 0.86767578125, -0.432373046875, -1.4716796875, -0.72998046875, 0.943359375, -1.9609375, 0.207275390625, -1.5400390625, 0.034332275390625, 0.66064453125, 0.20361328125, 0.032073974609375, 0.896484375, 0.9482421875, -0.302734375, -0.59033203125, 1.2138671875, -0.90625, -1.294921875, 0.40478515625, -0.68212890625, -0.96826171875, 1.990234375, -1.0712890625, -1.3232421875, 2.134765625, 0.216064453125, -2.189453125, 0.99755859375, -0.08953857421875, 0.57958984375, -0.336181640625, -0.071533203125, 0.234130859375, 0.81005859375, 0.05523681640625, -0.765625, -0.275390625, 1.8291015625, 0.94873046875, 1.03125, -0.57373046875, 0.239013671875, -0.892578125, 0.450439453125, -1.263671875, -0.20947265625, -0.452392578125, -0.1572265625, -0.8837890625, -0.9951171875, 0.318115234375, 1.126953125, -0.358154296875, -1.279296875, -0.399169921875, 0.9365234375, -0.4453125, -1.220703125, 1.1337890625, 1.0576171875, 1.71875, -0.89990234375, -0.73291015625, -2.025390625, 0.7041015625, -0.91796875, 0.00872039794921875, -0.7197265625, -0.859375, -1.35546875, 0.0294647216796875, -0.70068359375, 1.7734375, -1.6201171875, -0.7138671875, 0.0197296142578125, -0.2080078125, 0.98486328125, -0.99072265625, 0.0941162109375, -0.146728515625, 1.1708984375, -0.436279296875, -0.5048828125, 0.72607421875, 0.282470703125, -1.5673828125, -0.6806640625, -1.2998046875, -1.0703125, -0.439453125, -0.477783203125, -1.3046875, -1.0234375, 2.048828125, 0.9423828125, -1.1787109375, -0.292724609375, 0.197509765625, -0.2227783203125, 0.732421875, 1.8486328125, 0.276123046875, 1.6025390625, 0.335205078125, 0.7109375, -2.310546875, -0.463134765625, 0.2705078125, -1.3203125, -0.348388671875, -1.76171875, 0.62841796875, 1.4580078125, 0.249267578125, 1.2568359375, -1.2119140625, -1.4580078125, 1.298828125, -1.185546875, 0.5224609375, -0.1875, -0.5634765625, -0.01441192626953125, -0.189697265625, -0.2646484375, 1.1982421875, -2.103515625, -0.311279296875, 1.638671875, -3.203125, -0.6455078125, -0.257080078125, 1.8076171875, 0.12158203125, 0.34228515625, -0.00920867919921875, -0.49609375, -0.517578125, 1.6796875, -1.6279296875, 0.11578369140625, 0.0127105712890625, -0.170654296875, 1.4677734375, -1.3544921875, 0.166748046875, 0.6318359375, 0.81103515625, -0.84716796875, -0.599609375, -0.39208984375, 0.63134765625, 0.734375, -0.364990234375, -0.98583984375, 0.18212890625, 0.78125, 0.84375, 0.08099365234375, 0.70263671875, -1.4443359375, 0.63330078125, -0.133056640625, 0.79150390625, 0.10137939453125, 1.0087890625, -0.1654052734375, -1.4384765625, -0.1051025390625, -0.483642578125, -1.3115234375, -0.2415771484375, -0.53662109375, 0.323486328125, 1.708984375, -0.361572265625, 0.034820556640625, 0.0789794921875, 1.5, -0.319580078125, 1.4091796875, 1.6083984375, 0.98046875, -1.3232421875, -0.60888671875, -0.07080078125, -2.056640625, 0.69580078125, -1.986328125, 0.638671875, -1.0830078125, 0.37158203125, -2.416015625, -0.0762939453125, 1.2587890625, 1.9140625, -2.419921875, -0.3935546875, 0.763671875, 1.08203125, -0.10302734375, -0.89306640625, -0.79345703125, 0.919921875, 0.25439453125, -1.1708984375, 0.591796875, -1.859375, 0.428466796875, -0.931640625, -0.72900390625, -1.29296875, 1.2265625, 0.249267578125, -0.01934814453125, -0.345458984375, -1.095703125, -0.480224609375, -1.4052734375, -0.252197265625, 0.1798095703125, -0.54443359375, -0.63818359375, -0.63037109375, -0.2281494140625, 1.7685546875, -0.74853515625, -0.85400390625, 0.4033203125, 0.426513671875, -0.63134765625, -1.533203125, -2.3125, -1.2685546875, 0.974609375, -0.423583984375, -1.509765625, 0.5673828125, -0.349853515625, -0.86669921875, -0.90478515625, 0.303466796875, -1.6083984375, -2.505859375, 1.0322265625, -0.38427734375, 0.417724609375, -1.1396484375, -1.3251953125, -2.265625, 0.05523681640625, -1.5283203125, -0.2548828125, 0.62890625, -1.7734375, -0.47509765625, 0.51416015625, -0.462158203125, -0.12890625, -0.374267578125, 1.2578125, 0.6787109375, -0.2415771484375, 0.87939453125, -0.116455078125, -1.00390625, -0.73095703125, -0.84423828125, 0.396484375, -0.9482421875, 0.54345703125, 1.162109375, -1.052734375, -0.703125, -0.84326171875, 0.60595703125, -0.343505859375, -1.3505859375, 1.728515625, -1.2841796875, -0.2281494140625, 0.2587890625, 0.392578125, 0.036590576171875, -2.154296875, 0.0745849609375, -0.05743408203125, -0.9189453125, 1.7509765625, -0.4443359375, 1.2060546875, 0.521484375, 0.6748046875, 0.9853515625, -0.55810546875, 0.994140625, 1.40234375, 2.359375, -1.6181640625, 0.07196044921875, -0.67333984375, 3.26171875, 2.3203125, -2.263671875, -0.83740234375, 1.294921875, -0.06317138671875, -0.3486328125, -0.49365234375, 0.92529296875, 0.65478515625, 0.6796875, -0.4375, -0.408935546875, 1.1162109375, -0.481689453125, -0.140625, 0.94873046875, -1.2451171875, -0.8125, 0.59619140625, -0.158447265625, -0.29736328125, -0.484130859375, -0.58251953125, 2.4140625, 0.51416015625, -0.91455078125, -0.360107421875, 0.51025390625, -0.8037109375, 0.6806640625, 2.60546875, 0.96142578125, -1.0947265625, 0.56787109375, 0.380126953125, -0.336181640625, 0.218505859375, -1.4638671875, -0.55615234375, -0.80517578125, 1.443359375, 0.45361328125, -0.1298828125, 0.228271484375, 0.89794921875, -0.2822265625, -2.15625, 0.00807952880859375, -0.50390625, 0.736328125, 0.55419921875, -0.429443359375, 0.1942138671875, 0.36865234375, -1.21875, 0.54541015625, 0.1685791015625, 0.42431640625, 0.363525390625, -0.70166015625, 0.59228515625, 1.6064453125, 0.6025390625, 0.7646484375, 0.33447265625, 0.5126953125, 1.287109375, 1.380859375, -0.177490234375, -1.1884765625, 0.74365234375, -0.54931640625, -2.40625, 0.134765625, 2.349609375, 1.056640625, 0.72607421875, 1.5703125, -0.69970703125, -0.70068359375, 0.476806640625, 0.9775390625, 2.3984375, -1.0263671875, -0.53466796875, -2.171875, -0.2275390625, 0.5908203125, 1.0673828125, 1.21484375, 0.406005859375, -0.2235107421875, -1.1298828125, -0.435791015625, 0.302978515625, -0.3193359375, 1.0341796875, -0.01540374755859375, 0.2442626953125, -2.087890625, -0.41064453125, 0.8955078125, -0.385009765625, -0.94921875, -0.14208984375, -1.8681640625, -0.68896484375, -0.6591796875, -1.1005859375, 0.78857421875, -0.27880859375, 1.8095703125, 0.0694580078125, -1.021484375, -0.609375, 0.447265625, -1.7294921875, 0.03076171875, 2.91015625, 0.94775390625, -1.255859375, 0.08599853515625, -1.1884765625, -1.34765625, 1.2109375, -1.4453125, -0.22802734375, 0.04571533203125, -2.30078125, 0.8154296875, 0.966796875, 2.462890625, 0.98974609375, -0.60302734375, 1.3515625, 0.8408203125, -0.356689453125, 1.615234375, -0.2626953125, 0.1446533203125, 0.34130859375, -0.3076171875, -0.0306549072265625, 0.4736328125, 0.1297607421875, -1.138671875, 0.03369140625, -1.076171875, 1.0419921875, 0.58740234375, 1.875, 1.052734375, 0.307861328125, -1.123046875, -0.33056640625, 0.031951904296875, -1.763671875, 1.2548828125, 0.245849609375, -0.57666015625, -1.31640625, 1.208984375, -0.75927734375, -0.64111328125, -1.451171875, 1.4599609375, -0.3603515625, -0.8759765625, -1.6845703125, -1.552734375, -1.4873046875, -0.178955078125, 0.912109375, 2.447265625, -0.5224609375, 0.302001953125, -0.495849609375, 0.33935546875, -1.744140625, 1.71484375, -1.76171875, 1.0576171875, 1.2607421875, -0.2030029296875, -0.181396484375, 1.109375, 0.74609375, -0.68017578125, -0.2685546875, 1.490234375, 1.5380859375, 0.2030029296875, -0.9658203125, 0.810546875, -0.97705078125, -0.09747314453125, 0.69970703125, -1.0029296875, -1.65234375, -0.5166015625, -2.0703125, 1.1767578125, 1.0927734375, 1.7431640625, 1.6884765625, 0.50634765625, 1.94140625, -0.248046875, 0.0232086181640625, 0.0245208740234375, -0.232177734375, 1.97265625, -0.0491943359375, 0.40234375, 0.533203125, 1.353515625, 1.125, 0.40087890625, 0.7412109375, -1.80859375, 0.99755859375, -0.5927734375, 0.1883544921875, -2.12109375, -0.81689453125, -1.296875, 0.0193023681640625, 0.348876953125, -0.27294921875, -0.88525390625, -1.41015625, 0.348876953125, 0.9248046875, -0.84765625, -0.59130859375, -0.85009765625, -1.0810546875, 1.044921875, -2.37890625, -0.82763671875, 0.132568359375, -0.3447265625, 1.5087890625, -2.63671875, 1.169921875, -0.552734375, 0.165771484375, -0.6572265625, -0.039459228515625, -0.16845703125, -0.9326171875, 0.56201171875, 0.748046875, -1.3388671875, 0.10723876953125, -0.841796875, -0.1474609375, 0.734375, 1.1884765625, 0.31787109375, 0.086669921875, 0.29443359375, 0.238525390625, 0.2252197265625, -1.533203125, 1.6806640625, -0.1549072265625, -0.7353515625, 0.861328125, -0.50048828125, 0.10302734375, 0.27978515625, -1.05859375, -0.35107421875, 0.03692626953125, -1.806640625, 0.439697265625, -0.71240234375, -0.9033203125, -1.5244140625, 1.7646484375, 1.822265625, -1.6015625, 1.021484375, 0.78076171875, 2.306640625, -0.0606689453125, -0.08551025390625, 1.0908203125, 2.06640625, 0.149169921875, 0.3232421875, 0.0540771484375, -0.8583984375, 0.465576171875, -0.37060546875, -0.7978515625, 1.4814453125, -0.19580078125, 0.036651611328125, -0.91357421875, -0.19091796875, 1.205078125, -1.4951171875, -1.0771484375, 0.015655517578125, 0.3828125, 1.048828125, 0.281494140625, -0.6552734375, -0.60986328125, 0.317138671875, 0.90185546875, -0.615234375, 0.57177734375, 0.102294921875, 0.72607421875, 1.0498046875, -1.54296875, 1.271484375, 0.386474609375, -0.75146484375, -0.458984375, -0.0469970703125, -1.0927734375, 0.814453125, -0.86865234375, 0.4189453125, 2.232421875, -0.2003173828125, 0.87451171875, 1.4853515625, -0.40625, 0.466796875, 0.4521484375, 0.45947265625, 0.826171875, -0.6865234375, -1.3828125, -0.0169830322265625, 2.21484375, 0.63427734375, 0.004726409912109375, -0.2484130859375, 0.09765625, -0.416259765625, 0.73046875, -0.456298828125, 0.4873046875, 0.438720703125, -0.406494140625, -1.83203125, -1.4091796875, 0.9345703125, -1.3642578125, -0.51123046875, 1.1376953125, 1.369140625, -0.1846923828125, -2.580078125, 1.7431640625, 0.32568359375, 0.41064453125, -0.91455078125, 0.95263671875, 1.4521484375, -1.8525390625, 0.338134765625, 0.1429443359375, 1.18359375, 0.12408447265625, -2.138671875, -1.0478515625, 1.2080078125, 0.88037109375, -0.54736328125, 1.7958984375, -0.1953125, -0.237548828125, -0.0328369140625, -1.5478515625, 0.9541015625, -0.68896484375, -0.4794921875, 0.0260772705078125, 1.2080078125, 0.744140625, -1.0888671875, 0.344482421875, 0.6572265625, -0.1488037109375, 0.8408203125, -0.2266845703125, 1.3173828125, -0.342041015625, 1.8681640625, 0.8447265625, 0.27734375, -0.6044921875, -1.6240234375, -0.047119140625, 1.5341796875, -1.638671875, -0.496337890625, -0.4345703125, -0.677734375, 0.3876953125, 0.53076171875, 0.865234375, -1.2626953125, 0.59423828125, 1.376953125, -1.126953125, -0.521484375, 0.6826171875, 1.798828125, -1.7578125, 1.5517578125, -0.8330078125, -0.1539306640625, -1.2587890625, -0.381103515625, -1.244140625, -1.61328125, -1.1474609375, -0.324462890625, 0.2484130859375, 1.6962890625, -1.6201171875, 0.8359375, -0.41455078125, -0.86962890625, -0.26220703125, -0.8212890625, -0.11102294921875, 1.1826171875, -0.5576171875, 0.583984375, -1.3115234375, -2.578125, 0.10003662109375, 2.044921875, 0.34521484375, 0.5302734375, 0.449951171875, 2.59375, 0.6865234375, -1.7080078125, 0.1497802734375, -0.04901123046875, 1.740234375, -0.072265625, 1.029296875, -0.89306640625, -1.2978515625, -0.5966796875, 0.5029296875, -0.619140625, -0.962890625, -0.6494140625, -1.03515625, -0.1832275390625, -0.353759765625, -0.57470703125, -1.4404296875, 0.08526611328125, -0.0112762451171875, 0.81298828125, 1.107421875, -0.82470703125, -1.794921875, -1.4189453125, 0.5302734375, -0.135498046875, -0.4541015625, -0.023406982421875, -0.609375, -0.05108642578125, 0.541015625, -0.30615234375, -0.91943359375, -1.69140625, -0.55859375, 0.463623046875, -0.62451171875, -1.26953125, -0.345703125, -0.93896484375, -0.5751953125, 1.07421875, -0.64990234375, 0.24560546875, 1.4375, -0.2457275390625, -0.5751953125, -0.33984375, 1.138671875, 0.2039794921875, -0.2841796875, -2.134765625, -0.295166015625, -1.2421875, 0.14501953125, 0.036865234375, 0.053955078125, -1.3720703125, 1.4580078125, -1.2978515625, 0.347900390625, -0.313232421875, 0.452880859375, 0.197998046875, -0.308349609375, -0.52490234375, 0.59423828125, 1.1630859375, -0.80419921875, -0.76953125, -0.131591796875, -0.52783203125, 0.49755859375, -0.365234375, 0.5927734375, 0.75439453125, 0.41748046875, -1.2822265625, 0.95068359375, -1.42578125, 1.3505859375, 0.07275390625, -0.32470703125, -1.416015625, 0.67919921875, -0.461669921875, 0.430419921875, -0.1651611328125, -0.379638671875, -0.8203125, -0.8505859375, 1.30078125, -1.09765625, -0.8369140625, 0.97314453125, 0.1324462890625, 1.3603515625, -0.3515625, 0.17919921875, 0.333984375, -1.0751953125, 1.185546875, 1.65625, -0.28955078125, -1.1533203125, 1.1728515625, 0.2919921875, 1.3798828125, 0.8046875, -0.31591796875, -0.97021484375, 0.55078125, -0.1192626953125, -1.3203125, -0.379150390625, 0.9775390625, -3.814453125, 0.25732421875, -0.99658203125, 0.212158203125, -0.78564453125, -0.814453125, 0.38818359375, 3.185546875, 1.1357421875, 0.45458984375, -1.5126953125, -1.1044921875, 0.1654052734375, -0.436767578125, 1.6455078125, -0.297119140625, 0.401123046875, -0.057342529296875, -0.61572265625, -0.1263427734375, 0.55517578125, 1.953125, 0.3994140625, -0.47900390625, -0.279296875, -0.280517578125, 1.3359375, -1.529296875, 0.273193359375, 1.884765625, 0.82470703125, -1.4580078125, -0.59423828125, 1.3427734375, -0.69775390625, -0.040069580078125, -0.91259765625, -0.5712890625, -0.1834716796875, 0.642578125, -0.386474609375, 0.5771484375, 0.10577392578125, 0.6484375, 0.010772705078125, -1.896484375, -0.48388671875, -0.4873046875, 1.0703125, 0.1627197265625, -0.29296875, 0.8154296875, 0.0606689453125, 1.6142578125, 0.474853515625, -0.263427734375, -0.888671875, 0.279541015625, 0.1318359375, -0.1424560546875, -0.7353515625, 1.369140625, 0.490234375, 0.79296875, 1.9599609375, 0.375244140625, -2.271484375, -0.552734375, 2.83984375, -0.259521484375, -0.038543701171875, 0.1319580078125, -0.0231781005859375, 1.9736328125, -0.8720703125, -1.01953125, -0.37744140625, -1.369140625, -0.5478515625, 1.46484375, -0.53125, -1.0830078125, 0.089111328125, -1.662109375, 0.7216796875, 1.8505859375, -0.12249755859375, -0.10748291015625, 0.86962890625, 0.9482421875, -0.85986328125, 0.76025390625, -1.7666015625, -1.4521484375, 0.580078125, -2.119140625, -0.52978515625, -0.10589599609375, 0.9091796875, 1.76953125, -0.6552734375, -0.284912109375, -0.2861328125, 0.7451171875, -0.439697265625, -1.06640625, -0.191162109375, 0.9111328125, 0.17822265625, -1.13671875, 0.140625, -0.861328125, 1.2744140625, 1.0107421875, 1.3828125, 2.1015625, 0.25341796875, 0.64501953125, 1.1474609375, -0.8515625, -1.1279296875, 0.1705322265625, -1.1796875, 0.15869140625, 0.7041015625, 0.9033203125, 0.50048828125, 0.70556640625, -0.25537109375, 0.09130859375, 0.830078125, -0.8955078125, 0.833984375, 0.278076171875, -2.740234375, 1.6533203125, 0.71728515625, 0.449951171875, -0.0794677734375, -0.080322265625, -1.458984375, -0.3955078125, 0.466796875, 0.05877685546875, 1.0107421875, -0.43115234375, 0.1748046875, 0.2236328125, 0.8349609375, -1.13671875, -0.93310546875, 1.0546875, -0.075927734375, 0.62109375, 0.374755859375, -0.361083984375, 0.94482421875, 1.00390625, 0.359375, 1.15625, 0.1856689453125, 0.57275390625, 1.20703125, -1.0712890625, 0.6259765625, 0.95166015625, -0.16015625, 2.078125, 0.9560546875, -1.3720703125, 0.49755859375, 1.068359375, -0.77001953125, 1.51171875, -1.1103515625, 0.25830078125, 1.017578125, -0.10693359375, -0.56298828125, 0.1396484375, 0.06866455078125, 0.89892578125, 0.36962890625, -0.1531982421875, -0.2158203125, -0.1888427734375, -0.91845703125, 0.51123046875, 1.091796875, 1.623046875, -0.92529296875, -0.35595703125, 0.369873046875, 0.61669921875, -0.03167724609375, -0.9150390625, 0.60595703125, -0.206787109375, -0.70947265625, -0.80712890625, -2.0078125, -0.0823974609375, 0.9521484375, 0.1014404296875, -0.339599609375, 0.81884765625, 1.396484375, 0.2080078125, -0.94482421875, -0.437255859375, -0.322265625, 0.4130859375, 0.8828125, 0.0716552734375, -1.2509765625, -1.40234375, 0.830078125, 0.3974609375, -0.8857421875, -0.431396484375, -0.461181640625, 1.287109375, -0.6494140625, -0.25390625, -1.6064453125, -1.931640625, -0.23388671875, -1.1640625, 1.1513671875, 0.88330078125, 2.10546875, 0.61181640625, 0.34521484375, 0.56884765625, -0.61962890625, -1.060546875, -0.00884246826171875, -0.2174072265625, 1.216796875, -0.58447265625, 0.7021484375, -1.5849609375, 0.2252197265625, -0.342529296875, -0.52734375, -0.0118255615234375, 0.007099151611328125, 0.818359375, -0.046478271484375, -0.13623046875, -0.8984375, 0.07666015625, -0.818359375, -0.0841064453125, 0.126953125, -1.4921875, -0.85009765625, -0.31982421875, 0.15380859375, -1.41015625, -1.677734375, -1.7490234375, 0.1641845703125, -2.076171875, 0.001708984375, -0.31591796875, -0.7822265625, -0.7314453125, 1.53515625, 1.158203125, 1.48828125, -0.5693359375, -1.2978515625, -1.1533203125, -0.81640625, -1.017578125, -0.51171875, -0.31591796875, -1.861328125, 0.377685546875, 0.214111328125, -0.84130859375, -1.2412109375, -1.609375, 1.5908203125, -0.321044921875, 0.8232421875, -0.033599853515625, -0.72607421875, 0.12493896484375, 1.37890625, -0.78662109375, 3.029296875, 1.54296875, 0.383544921875, 1.0517578125, -0.1629638671875, 0.150390625, -0.410888671875, 0.5205078125, 0.5390625, 0.037567138671875, 0.473876953125, -1.7236328125, 0.21826171875, 0.412109375, -1.65234375, 0.57275390625, 0.06890869140625, -1.4990234375, 1.0205078125, -1.0068359375, -0.46728515625, -0.86328125, 1.125, -0.66015625, -0.5810546875, 0.97998046875, -1.1005859375, -0.68603515625, -0.09832763671875, -0.370849609375, 0.49951171875, 0.47412109375, -1.8271484375, 0.79345703125, -1.1064453125, -0.172607421875, -0.0869140625, -0.818359375, 1.4208984375, -0.350341796875, -0.8359375, -0.306640625, 0.278076171875, 1.2177734375, 0.3740234375, 0.52978515625, 1.453125, 0.64404296875, 0.177734375, 1.3095703125, -1.0146484375, 2.375, 1.548828125, 1.140625, 1.4111328125, 0.89697265625, 1.7490234375, -1.421875, 0.78076171875, 1.1669921875, 0.348388671875, -0.2166748046875, 0.04840087890625, -0.65771484375, -1.4267578125, 0.099609375, -0.0136260986328125, 1.96875, -1.451171875, -0.09600830078125, 0.317626953125, 1.0771484375, -0.2440185546875, 2.126953125, -0.6748046875, -0.640625, 0.44140625, 0.94921875, 0.058990478515625, -0.96533203125, -1.1865234375, -0.876953125, -1.7841796875, 0.12890625, -0.058624267578125, -0.521484375, -0.1466064453125, 0.75439453125, 0.32421875, 1.6796875, -0.505859375, 0.376220703125, -1.244140625, 0.7158203125, -0.489501953125, -0.265380859375, 0.099853515625, 0.79345703125, -1.1337890625, -0.245361328125, 0.216552734375, -0.071044921875, 0.260009765625, -0.222900390625, -1.3134765625, -0.279296875, -1.171875, 0.0010499954223632812, -0.57470703125, 0.06280517578125, -0.48193359375, -0.89208984375, -0.048919677734375, -0.3330078125, 0.6142578125, -2.390625, -0.9208984375, -0.98486328125, -0.451171875, 0.61083984375, 0.15869140625, -0.389892578125, 0.361083984375, 0.8203125, -0.29248046875, 1.3759765625, 1.189453125, -0.370361328125, 0.6552734375, -1.2841796875, 0.1546630859375, -0.59521484375, 1.0048828125, 0.521484375, -1.0654296875, -0.533203125, 2.4765625, -0.51904296875, 0.69091796875, 1.025390625, -0.0916748046875, 0.2381591796875, 1.3076171875, 0.91064453125, 1.0087890625, -2.2578125, 1.21875, 1.1962890625, -1.072265625, 0.51025390625, 0.0248565673828125, -0.08905029296875, 0.9443359375, 1.619140625, 2.599609375, 0.21240234375, 3.240234375, 0.9443359375, -1.0732421875, 1.2431640625, -0.475830078125, 0.54052734375, 1.58984375, -0.169921875, 0.6015625, -0.3623046875, -1.896484375, -0.041717529296875, 0.0845947265625, -0.67333984375, 1.0263671875, 1.568359375, 0.0662841796875, -0.80712890625, -1.228515625, -1.380859375, -1.197265625, -0.9599609375, 0.63427734375, -0.360595703125, 0.51416015625, 0.69873046875, 0.96044921875, 0.0531005859375, -1.0068359375, 1.2705078125, -0.06866455078125, 0.11273193359375, 0.94677734375, 0.057830810546875, -1.4140625, -0.1241455078125, 0.059326171875, -0.0183258056640625, -0.5068359375, 2.32421875, -1.1279296875, 0.32421875, -0.56298828125, 0.35791015625, 0.185791015625, -0.990234375, 0.1396484375, 0.63525390625, -0.79443359375, -1.0498046875, 0.08831787109375, 1.4072265625, 0.89013671875, -0.736328125, 0.0321044921875, -0.984375, -0.0085906982421875, -1.2822265625, -0.75390625, 0.2249755859375, 0.63427734375, -1.3017578125, 0.81103515625, -0.331787109375, -1.3125, 2.51953125, -0.290771484375, -0.74462890625, -0.05889892578125, 1.32421875, 0.884765625, 0.4296875, 1.2978515625, 0.78271484375, 1.0849609375, 0.69482421875, 2.392578125, 0.89501953125, 1.4873046875, 0.302734375, -0.135498046875, -1.5419921875, 0.10711669921875, 0.61474609375, 0.314697265625, -0.20849609375, -1.462890625, -0.111328125, 0.237060546875, -0.5810546875, 1.4091796875, -0.42138671875, -1.671875, -0.56689453125, -1.525390625, 0.379638671875, -0.00022041797637939453, -0.2335205078125, -0.197021484375, -0.29345703125, -1.8798828125, 0.442626953125, -1.490234375, -1.5576171875, 1.560546875, 0.53271484375, 1.8935546875, -1.0390625, -1.01171875, -0.54248046875, -0.33984375, -0.2239990234375, 0.404296875, 1.0087890625, -0.84619140625, 0.43603515625, -0.5, 0.1357421875, -0.56640625, -0.962890625, 1.5361328125, 0.034576416015625, 0.10125732421875, 0.53173828125, -0.197509765625, 0.2275390625, 2.111328125, 0.05078125, -0.78857421875, 0.68505859375, 0.391357421875, -0.623046875, -1.3095703125, 1.0107421875, -0.689453125, 0.62744140625, -0.8681640625, 0.268798828125, 1.3857421875, -0.28662109375, -0.41796875, 0.141845703125, -0.14501953125, -1.19140625, -0.63818359375, 0.55712890625, 1.34765625, -1.4580078125, -0.388671875, 2.31640625, 0.8916015625, -1.2646484375, 1.060546875, 2.845703125, 0.1234130859375, 0.65283203125, 0.923828125, -0.572265625, 0.037109375, 0.86474609375, -1.2001953125, -0.1673583984375, 0.34423828125, 2.541015625, -0.44921875, 1.0322265625, -1.0244140625, 1.203125, -0.74951171875, 0.46826171875, 1.40234375, 0.329833984375, -1.6318359375, -1.2919921875, -1.126953125, 1.69921875, 0.1356201171875, -0.11346435546875, 1.400390625, 1.3046875, 0.81884765625, -2.423828125, -0.0083770751953125, 0.54443359375, 0.86083984375, 0.182861328125, -0.0587158203125, 0.93701171875, 1.2578125, 1.2333984375, -0.117919921875, -1.580078125, -0.802734375, -1.0546875, -0.14404296875, -1.193359375, -0.79931640625, -0.96728515625, 0.697265625, 1.048828125, -0.6435546875, -1.611328125, -0.37353515625, 0.8759765625, 1.615234375, 0.08807373046875, 1.4580078125, 1.36328125, -1.02734375, -1.5009765625, 0.560546875, -2.04296875, -1.3671875, 2.341796875, 0.5869140625, -2.171875, 0.200439453125, 1.9873046875, -1.5244140625, -0.84423828125, 0.68994140625, -1.078125, 1.107421875, -0.6201171875, 0.73876953125, -0.7626953125, 0.00617218017578125, 0.337890625, -0.494873046875, -0.91943359375, 0.470703125, -0.19775390625, -0.176513671875, 0.7646484375, 1.4541015625, 0.285400390625, -1.1357421875, 1.1689453125, -0.72802734375, -2.69140625, 0.767578125, -1.0048828125, -0.378662109375, 1.8798828125, -1.5537109375, 0.71533203125, 0.60791015625, 0.96875, -0.063720703125, 0.623046875, -0.71533203125, -0.5947265625, 1.3984375, -0.0227203369140625, -1.46875, 0.37451171875, 1.9560546875, 0.084228515625, -0.394287109375, 1.1123046875, 0.1026611328125, -0.480712890625, 0.93505859375, 1.818359375, 0.9130859375, -0.7900390625, 1.1328125, 0.80029296875, -0.253662109375, -0.069580078125, -0.58203125, 1.1669921875, 0.356201171875, 0.9287109375, 0.212158203125, -0.0867919921875, 0.35009765625, 0.85693359375, -0.2109375, -1.8779296875, 0.251220703125, -1.1220703125, 0.245361328125, 0.60791015625, 0.998046875, 0.244873046875, 1.150390625, -1.2177734375, 0.638671875, 2.1953125, 0.177490234375, -0.96826171875, 0.288818359375, 0.826171875, 0.04840087890625, 0.2021484375, -0.74658203125, 0.5625, -0.1650390625, -0.7431640625, -0.78173828125, -1.2275390625, 0.646484375, 2.41796875, -0.078369140625, -1.3642578125, 0.075927734375, 0.09759521484375, -1.6318359375, -0.00856781005859375, -0.358154296875, 0.931640625, 0.2061767578125, 1.06640625, -0.921875, 0.2303466796875, -0.49951171875, 1.09765625, -0.204833984375, -0.4921875, 1.5712890625, -0.84228515625, 1.1298828125, -0.243896484375, -0.1204833984375, -0.09136962890625, 0.19873046875, -1.5361328125, -1.390625, 0.158447265625, -1.0537109375, 0.06494140625, 0.347900390625, 0.288330078125, -0.65380859375, 1.572265625, -0.8427734375, -0.220458984375, -1.9072265625, 1.77734375, -1.0400390625, 0.1263427734375, 0.09521484375, -0.71826171875, -1.720703125, -0.2003173828125, -0.05133056640625, -0.58642578125, -0.2386474609375, -1.3662109375, 0.2247314453125, 0.072021484375, -0.30029296875, 0.80810546875, -0.8740234375, -0.97314453125, -0.4169921875, -1.095703125, 1.962890625, 1.1767578125, 0.49072265625, -0.74462890625, 0.931640625, -0.57666015625, -0.3720703125, 1.1171875, -0.0703125, 0.1854248046875, -1.298828125, -0.18212890625, -0.024932861328125, 0.69091796875, 1.4208984375, 1.0146484375, 1.3037109375, 0.2274169921875, -1.1005859375, -0.701171875, -0.53662109375, 0.07708740234375, -0.57568359375, -0.8916015625, -2.02734375, -0.47509765625, 0.197265625, 0.423095703125, 1.5048828125, 0.2152099609375, -0.92138671875, -1.251953125, 0.1536865234375, -0.9365234375, 0.6328125, 0.37158203125, -0.402099609375, -0.67529296875, -1.935546875, -1.20703125, 0.97607421875, -0.76220703125, 0.775390625, 0.935546875, 0.845703125, -0.97509765625, 0.189208984375, -0.52783203125, -0.80712890625, 1.30859375, -0.08343505859375, -0.265380859375, 1.544921875, -0.41015625, 0.06304931640625, -1.4892578125, 0.7998046875, -0.11138916015625, -1.73828125, 0.09130859375, 0.25732421875, -1.267578125, 0.70703125, 0.6357421875, 1.2626953125, 0.54296875, 1.57421875, -0.86328125, -0.448974609375, 1.216796875, -1.7236328125, 0.1505126953125, -0.8857421875, 0.5849609375, -0.55615234375, 1.38671875, 0.599609375, -0.98388671875, -0.01462554931640625, -0.58056640625, 0.896484375, -0.440185546875, -1.0419921875, -0.60595703125, -0.873046875, -2.279296875, -0.1829833984375, -1.3876953125, -0.038787841796875, -1.1240234375, -0.0758056640625, 1.2763671875, -1.4931640625, 1.751953125, -0.17626953125, 2.267578125, -1.384765625, 1.3818359375, 0.859375, -0.31689453125, -0.95068359375, -0.499755859375, 1.8740234375, -2.26953125, -0.5830078125, -1.2265625, -1.33203125, -0.0809326171875, -0.97705078125, 0.440673828125, -0.7216796875, -1.064453125, 0.08837890625, 0.0279693603515625, 0.03192138671875, 1.001953125, 0.90673828125, 0.5234375, 0.84765625, 0.350830078125, 0.82177734375, 0.5673828125, 0.36376953125, -1.365234375, -1.603515625, -1.4296875, 0.58935546875, -1.607421875, -0.291259765625, -0.6640625, -0.62353515625, 0.4853515625, -1.1162109375, -0.74658203125, 0.469482421875, -0.912109375, 1.0078125, -1.0439453125, 0.923828125, 0.5166015625, -0.25244140625, 1.0634765625, 0.06329345703125, -0.165771484375, -0.464599609375, 0.1871337890625, 0.481689453125, -0.60107421875, -0.90234375, 1.5322265625, -1.2578125, -0.54443359375, 1.1123046875, 0.1536865234375, 1.166015625, 1.2666015625, -0.06549072265625, -0.5458984375, 0.84716796875, 2.23046875, 0.40673828125, -1.0322265625, 0.60107421875, 0.30322265625, -0.40673828125, 0.88720703125, -0.192138671875, -0.85693359375, -0.419677734375, 0.1392822265625, 0.59521484375, 0.094970703125, -0.281494140625, -0.72900390625, -0.7919921875, 0.36279296875, 0.134765625, -0.2147216796875, 1.7216796875, 0.58837890625, 2.0, 1.59765625, -0.908203125, -1.3642578125, 0.982421875, -0.47216796875, -0.131103515625, -0.51025390625, 0.53271484375, -0.064208984375, 1.6240234375, 0.73828125, -0.00862884521484375, 0.70458984375, 1.8857421875, 0.049835205078125, 2.248046875, 0.53125, 0.3427734375, 0.267333984375, -0.305908203125, -0.4189453125, -0.677734375, -2.435546875, -0.93310546875, 0.68115234375, -1.0693359375, -1.50390625, 0.73046875, -0.39599609375, -0.43994140625, 0.17236328125, -0.35400390625, 1.0380859375, 0.225341796875, -2.1484375, 0.5380859375, 0.66357421875, 0.059295654296875, -1.109375, -0.2939453125, -0.2607421875, -0.26123046875, 0.062042236328125, 0.38330078125, -0.7421875, 1.05859375, 0.55810546875, 0.1795654296875, -0.039642333984375, 0.705078125, 0.66064453125, -0.83642578125, 0.45458984375, 0.97119140625, 0.138671875, -1.2265625, 1.10546875, 0.7861328125, -0.87841796875, -0.75048828125, -0.1634521484375, -0.244140625, 1.376953125, -0.00745391845703125, -0.002643585205078125, 0.3916015625, 0.63330078125, 0.9794921875, 0.9912109375, -1.1201171875, 0.0948486328125, -0.66650390625, 1.1396484375, -1.65625, 0.161865234375, 0.6708984375, 0.51513671875, -0.23974609375, 0.923828125, 1.46484375, -0.6064453125, 0.146728515625, 1.1513671875, 0.428466796875, -0.4404296875, 0.3603515625, 0.15087890625, -0.75048828125, 0.1463623046875, -0.7509765625, 0.71826171875, -0.7197265625, -0.499267578125, 1.9736328125, -1.6044921875, -0.8056640625, -1.5146484375, 0.19384765625, -0.0018157958984375, 0.307861328125, -0.1417236328125, 0.51318359375, 0.312255859375, 0.83251953125, -1.12109375, 0.487548828125, 0.54345703125, -0.10980224609375, 0.27099609375, 0.1107177734375, 1.91796875, -1.4580078125, 0.5546875, -1.7099609375, -1.6767578125, -1.4404296875, -0.374755859375, -1.07421875, -0.4462890625, -1.681640625, -0.17333984375, 0.12286376953125, -2.37890625, 1.111328125, 0.0875244140625, 0.50537109375, 0.282958984375, 0.60791015625, -1.7802734375, 0.0248260498046875, -0.91748046875, -0.1954345703125, 0.132080078125, 1.9228515625, 0.5126953125, 0.892578125, 2.763671875, -1.095703125, -0.7294921875, -1.0966796875, 0.70947265625, -0.0836181640625, 1.0654296875, -0.74462890625, 0.330078125, -0.35205078125, 1.3515625, -1.63671875, 0.77197265625, 0.52294921875, -0.81884765625, 0.07257080078125, 0.54150390625, -1.0849609375, -2.14453125, 0.732421875, -1.42578125, -0.47021484375, -2.140625, -1.814453125, 0.424560546875, 0.2332763671875, 1.13671875, -0.038787841796875, 1.2998046875, 0.39013671875, -1.0185546875, 1.1943359375, 0.6142578125, 0.578125, 1.0947265625, -0.23779296875, 0.74169921875, -0.54296875, -1.7880859375, 0.84423828125, -0.8798828125, 1.9423828125, 0.119140625, 0.402099609375, -0.8427734375, -1.640625, -0.74853515625, -0.181396484375, 0.7373046875, -0.1219482421875, -1.0537109375, -2.001953125, 0.99951171875, -1.8984375, 0.8408203125, -2.58984375, -0.6806640625, 1.3154296875, 0.427734375, 0.271240234375, -0.180908203125, -1.6025390625, 0.08544921875, 0.8671875, 0.323974609375, 1.654296875, -1.833984375, -0.7900390625, 0.09393310546875, 2.69921875, 0.399658203125, 0.476806640625, 1.2939453125, -0.38330078125, -1.744140625, 0.06329345703125, 0.83837890625, 0.09368896484375, 1.04296875, 1.763671875, 0.9287109375, -0.20849609375, 0.005741119384765625, 0.3779296875, 1.0029296875, 0.625, -0.689453125, 0.4306640625, -0.011688232421875, 1.0341796875, -0.427001953125, 1.056640625, -0.59521484375, -0.9541015625, -0.6435546875, -0.84814453125, -0.474609375, -1.17578125, 0.71435546875, 0.07049560546875, 2.119140625, -1.0166015625, 0.360107421875, -2.24609375, -0.34619140625, 0.6005859375, 1.3232421875, -0.095458984375, -1.5498046875, -0.87158203125, -1.2314453125, 0.087890625, 0.418701171875, -0.411865234375, 1.23046875, 0.8544921875, -1.2783203125, 0.685546875, 0.75048828125, 0.265869140625, -1.6357421875, -0.3955078125, -0.485107421875, 0.826171875, 0.14697265625, -0.0220184326171875, -0.1917724609375, -1.462890625, 2.45703125, 1.2490234375, -0.001338958740234375, 0.85009765625, 0.67138671875, -0.80712890625, 1.26953125, 1.2373046875, 0.65283203125, 0.5107421875, -0.1768798828125, -0.53076171875, -0.50146484375, -0.54833984375, 0.2763671875, 0.273681640625, 0.93896484375, 0.86669921875, -1.1884765625, 0.53515625, -0.052825927734375, -0.215087890625, -0.7236328125, -0.373779296875, -0.0728759765625, 0.80078125, -0.074951171875, 0.8583984375, 0.2032470703125, 2.919921875, 1.29296875, -0.68603515625, -0.654296875, -0.876953125, 0.357421875, 0.20556640625, -1.095703125, 1.1396484375, 0.48291015625, -1.576171875, -1.0087890625, 1.3046875, 0.35595703125, 0.356689453125, -0.311279296875, -0.42236328125, -0.6923828125, 2.486328125, 1.068359375, 1.1162109375, 0.63623046875, -0.46923828125, 0.2467041015625, 0.69287109375, -0.6953125, 0.46240234375, 0.54541015625, -0.297119140625, 0.6943359375, -0.4658203125, -0.054901123046875, 0.483154296875, -0.24853515625, 0.424560546875, -1.9013671875, 1.8369140625, 0.8291015625, -0.82080078125, 0.2427978515625, -0.5166015625, 1.7890625, 0.8291015625, 0.51416015625, 0.6953125, -1.2470703125, -0.806640625, 1.2958984375, -1.4189453125, -0.5771484375, 0.56689453125, -0.138671875, -1.1689453125, -0.68212890625, -0.42919921875, -1.0166015625, 0.121337890625, -0.6806640625, -0.2447509765625, 0.50634765625, 0.8408203125, 1.23828125, 1.4599609375, -0.12841796875, 0.1634521484375, 0.1275634765625, -0.232177734375, 0.051300048828125, -0.11761474609375, 0.1920166015625, 0.466552734375, 0.7958984375, -0.6337890625, 1.427734375, 1.1015625, -0.358154296875, -1.6201171875, -1.103515625, 1.97265625, -0.611328125, 0.9482421875, -0.2459716796875, -1.1171875, -0.8115234375, 0.03668212890625, 0.1376953125, 0.26318359375, -0.79296875, 0.434814453125, 0.962890625, -0.88720703125, -0.1793212890625, -0.54638671875, 1.4150390625, 0.233642578125, -1.390625, 0.2093505859375, -1.951171875, 1.5126953125, 0.541015625, -0.021514892578125, 0.8779296875, 1.8427734375, -0.10162353515625, 0.181884765625, 0.11981201171875, 2.08984375, -0.72314453125, -1.4501953125, -0.62890625, -0.08758544921875, 0.12744140625, -0.377685546875, 1.650390625, 2.369140625, 1.2255859375, 0.0614013671875, 0.4404296875, 0.10888671875, 1.810546875, -2.041015625, 1.771484375, 0.7646484375, -0.64892578125, -0.84033203125, -0.70703125, 0.7607421875, 1.490234375, -2.35546875, 0.156494140625, -0.591796875, -1.4169921875, -1.884765625, -0.2296142578125, -1.2236328125, 0.64892578125, 0.462158203125, -1.6162109375, -0.2919921875, 1.65234375, 0.9326171875, 0.8994140625, 1.841796875, 0.9580078125, 0.9150390625, -0.18212890625, 1.8984375, -0.11517333984375, 0.56494140625, -1.3603515625, -1.1591796875, -0.884765625, -0.4697265625, 1.1455078125, -1.3642578125, 0.60498046875, 0.39599609375, -0.1260986328125, 0.058135986328125, 1.5107421875, 0.84716796875, 1.486328125, 1.0732421875, -0.8095703125, -0.29833984375, -0.238525390625, -0.689453125, -0.556640625, 0.005290985107421875, 0.80517578125, -0.28857421875, -0.043121337890625, -0.296142578125, 1.787109375, 1.125, 0.486083984375, 0.3798828125, -0.356689453125, -0.9775390625, -0.958984375, 1.0859375, 0.5048828125, 0.08880615234375, 0.5947265625, -1.4697265625, 1.595703125, 0.09320068359375, -0.40283203125, -0.041259765625, -2.2734375, -0.242919921875, -0.65283203125, 1.7783203125, -2.091796875, -0.00783538818359375, -0.74658203125, -0.18408203125, -0.27294921875, 1.86328125, -1.21484375, -0.6181640625, 1.4375, -0.65576171875, 1.2509765625, 0.07891845703125, -0.4287109375, -0.3486328125, 0.1446533203125, -0.51318359375, -1.548828125, -1.5234375, 0.71728515625, -2.228515625, 0.904296875, 0.2059326171875, -0.08172607421875, 0.373779296875, -0.474853515625, 0.490966796875, 1.083984375, 0.92626953125, 0.8505859375, -1.005859375, 0.06524658203125, -0.2498779296875, -0.71533203125, -0.04925537109375, 1.462890625, -0.88330078125, 1.2421875, -0.337158203125, 0.1959228515625, 0.6533203125, -1.2939453125, 1.447265625, -1.236328125, -0.68212890625, -1.724609375, 1.2138671875, 0.611328125, 0.66650390625, 0.50830078125, 0.047027587890625, 0.68505859375, -0.51025390625, -0.61279296875, -1.04296875, -0.6181640625, -0.31103515625, -0.96533203125, -0.9521484375, 0.86962890625, -0.4462890625, 0.0154266357421875, 0.68115234375, -0.51806640625, -0.99267578125, -1.4287109375, -0.5810546875, -1.857421875, -1.068359375, -0.89990234375, -0.324462890625, -0.359375, 0.423095703125, 0.7900390625, -0.301513671875, -0.70458984375, -0.654296875, -0.89599609375, -2.12890625, 0.66650390625, 0.04254150390625, -0.293701171875, -0.1661376953125, -1.4833984375, 0.66650390625, -0.560546875, 0.7578125, -0.5244140625, -0.1617431640625, 0.88916015625, 0.456298828125, 1.1162109375, 0.5419921875, -0.3681640625, -1.12890625, -0.30078125, -0.88134765625, 0.93701171875, -0.5595703125, -0.06396484375, -0.79296875, 0.0173492431640625, -2.490234375, 0.410888671875, 0.73828125, -0.615234375, 1.5498046875, -1.775390625, -1.17578125, 0.09002685546875, -0.740234375, -0.54052734375, 1.18359375, 1.4677734375, 2.35546875, -1.17578125, 0.1671142578125, 1.7265625, -0.302978515625, -0.055694580078125, 0.251220703125, 0.307373046875, -0.5556640625, -0.96142578125, 2.1953125, -0.79541015625, -0.93896484375, 1.1142578125, -0.244873046875, -1.53515625, -0.3095703125, -0.461669921875, 1.806640625, -0.0966796875, -0.17529296875, -0.2252197265625, -1.1953125, -0.82958984375, -1.03125, -0.126953125, 0.89208984375, 0.9365234375, -0.6650390625, -0.1773681640625, -1.1416015625, 0.9521484375, -0.2490234375, 0.66455078125, -0.04571533203125, 0.26708984375, -1.0322265625, 2.01171875, -0.357177734375, -0.321533203125, -0.324951171875, 0.71533203125, -0.52685546875, 1.7978515625, -0.3876953125, -0.3740234375, -0.03875732421875, 1.0634765625, 0.75048828125, -0.2156982421875, -0.755859375, -0.78125, -0.54443359375, -0.433349609375, -1.19921875, -0.029266357421875, -1.46484375, -0.92626953125, 0.142578125, -0.98193359375, -1.3603515625, -0.188720703125, -0.267822265625, -0.263916015625, -0.6044921875, -0.052215576171875, 0.1904296875, -0.22412109375, 2.427734375, 0.446044921875, 1.3642578125, 0.341796875, -1.72265625, -0.1282958984375, -1.51953125, -1.01171875, -1.375, -0.220458984375, -1.732421875, -1.8740234375, 0.6171875, 1.02734375, -0.173828125, -0.4716796875, 1.171875, 1.6044921875, 1.3466796875, 0.2010498046875, 0.86181640625, -0.56591796875, -0.80712890625, -0.79443359375, -0.81787109375, 0.1448974609375, -0.1689453125, -0.8818359375, 0.02899169921875, -0.73095703125, -0.353515625, -0.0889892578125, -1.763671875, 0.1568603515625, -0.51416015625, -0.9853515625, 1.0224609375, 0.2337646484375, 0.5703125, 0.1309814453125, -0.91064453125, 1.0625, -0.509765625, 0.998046875, -0.59130859375, 1.0947265625, -0.70361328125, -1.0078125, 0.11688232421875, -0.47021484375, -0.1842041015625, 0.85693359375, -1.0048828125, 0.71630859375, -0.11383056640625, -0.8955078125, -0.3447265625, 0.404541015625, 0.04656982421875, -0.283203125, 1.677734375, 0.160400390625, 0.42529296875, -0.088134765625, 0.95458984375, -2.12890625, -0.57958984375, -1.115234375, -0.5244140625, 0.28173828125, -0.85693359375, 1.517578125, 0.6572265625, -0.0753173828125, 0.56494140625, -0.63623046875, -0.488525390625, -1.4306640625, -0.57177734375, -1.0947265625, 1.0859375, 1.251953125, -0.2034912109375, 0.07904052734375, 0.72314453125, -1.6845703125, 0.8525390625, -0.07647705078125, -0.40234375, -0.8291015625, -0.61962890625, 1.2607421875, -1.064453125, -0.61572265625, 0.052001953125, 2.693359375, -0.833984375, 1.53125, 0.708984375, -0.59912109375, 1.009765625, -0.4853515625, -0.29443359375, 1.7041015625, 0.054901123046875, 0.2435302734375, 0.85595703125, -0.1810302734375, -1.4833984375, -1.6787109375, 0.226318359375, 1.7236328125, -0.44091796875, 1.7802734375, 1.05078125, 0.0869140625, -0.58935546875, -1.7431640625, -1.513671875, 1.1015625, 1.4072265625, 1.9326171875, -1.0908203125, 2.3125, 1.30078125, 2.603515625, 0.1470947265625, 0.90966796875, 1.591796875, -1.08984375, -1.0947265625, 0.5234375, -0.0242462158203125, -1.9130859375, 1.1826171875, -1.6279296875, 0.8271484375, -0.434814453125, -1.9033203125, -0.2423095703125, 0.8798828125, 0.56787109375, 1.7314453125, 2.318359375, -1.138671875, 0.55712890625, -2.91015625, -0.931640625, 0.099853515625, -0.896484375, 0.3173828125, -0.7509765625, 0.378662109375, 0.1551513671875, -1.0830078125, 0.357177734375, 0.494384765625, -0.5185546875, -0.02862548828125, -0.47216796875, 1.068359375, 1.0986328125, -2.1953125, 1.2529296875, -0.418212890625, 0.390625, -1.814453125, -0.029296875, -1.3154296875, 0.05743408203125, -0.09344482421875, 1.033203125, -1.1962890625, -0.2259521484375, -0.43896484375, 0.60986328125, -1.734375, 0.9130859375, -1.17578125, -0.398681640625, 0.11395263671875, 0.552734375, -0.578125, 0.37109375, -0.41455078125, 0.383544921875, 1.9755859375, -0.5595703125, -0.6767578125, -0.666015625, 1.765625, -0.375244140625, -0.33251953125, -0.376953125, 0.77001953125, -1.00390625, -0.342041015625, -0.11676025390625, 0.0306243896484375, 0.8115234375, 1.19140625, -0.1170654296875, -0.0968017578125, 1.1044921875, 0.2379150390625, 0.209228515625, 0.52294921875, 0.00739288330078125, 0.94873046875, -0.6298828125, -0.5126953125, 0.5087890625, 0.268310546875, 0.321533203125, -0.53125, -1.0380859375, 1.2275390625, -0.71435546875, -0.036651611328125, -1.16796875, 1.3173828125, 1.462890625, 2.29296875, 1.984375, 0.09686279296875, 0.58642578125, -0.11651611328125, -0.8671875, 0.69677734375, 0.336181640625, 0.564453125, 1.70703125, 0.57080078125, 0.250732421875, -2.36328125, 0.436279296875, 0.04547119140625, -1.3935546875, -0.1220703125, -0.10382080078125, 0.1905517578125, -0.68701171875, -2.62109375, -0.324462890625, -1.25, 1.30078125, -0.50439453125, -0.07916259765625, -0.09600830078125, 1.5859375, -1.2822265625, 0.50537109375, -0.56787109375, 1.125, 0.260009765625, -0.484619140625, -0.1663818359375, -2.287109375, -0.11517333984375, 1.2099609375, -0.392822265625, -0.43505859375, 0.51171875, -1.4521484375, 0.1884765625, 0.96142578125, 0.302490234375, -0.1839599609375, -1.08984375, -2.06640625, 0.625, 0.28466796875, -1.169921875, -0.54345703125, 0.51611328125, -1.203125, -0.0033397674560546875, 0.67626953125, 0.583984375, 2.396484375, 1.8330078125, -1.224609375, -1.953125, 0.537109375, -0.02960205078125, 2.03515625, 0.22509765625, -1.6396484375, -1.6064453125, -0.6923828125, 0.8291015625, 0.509765625, -1.8310546875, 0.385009765625, -1.03515625, -0.02911376953125, -1.7861328125, -1.333984375, -1.2373046875, -1.3046875, -0.043609619140625, 0.153076171875, 1.1533203125, -0.49365234375, 0.84130859375, -1.5478515625, -0.86572265625, 1.46875, -2.30859375, -0.5166015625, -1.244140625, 0.65771484375, -0.08001708984375, 0.8349609375, 1.8798828125, 0.767578125, 0.448974609375, -0.728515625, -0.459228515625, -0.379638671875, 0.339111328125, 0.02899169921875, -1.7587890625, 0.50537109375, -0.06787109375, -0.300537109375, 0.22998046875, -1.0458984375, -0.51123046875, 1.3837890625, 2.74609375, -0.77001953125, -0.12481689453125, 0.65478515625, 1.90234375, -0.1732177734375, -0.943359375, 1.4404296875, 0.4775390625, 0.432861328125, 1.669921875, 1.70703125, 0.0521240234375, 0.266845703125, -0.08831787109375, -0.759765625, 0.73876953125, 0.48095703125, 0.104248046875, 0.537109375, 0.5263671875, -0.75390625, -0.09393310546875, -0.9853515625, -1.37890625, 0.07196044921875, -0.62109375, 0.91845703125, 0.83251953125, -0.177490234375, 0.21044921875, 0.04669189453125, 2.103515625, 0.44189453125, -0.046844482421875, -1.0654296875, 0.421142578125, 0.98828125, -0.849609375, 1.4736328125, -2.1796875, -0.703125, 1.810546875, -1.84765625, 0.83154296875, 0.490966796875, -2.056640625, -0.50537109375, -2.447265625, -0.329345703125, -2.01953125, -0.84423828125, -0.83447265625, 0.6552734375, -0.90673828125, 1.376953125, 0.96337890625, 1.6064453125, -0.1866455078125, 0.59716796875, -0.77978515625, 1.2314453125, -0.88330078125, 0.98828125, 0.98876953125, 1.646484375, 1.650390625, -1.3115234375, 1.447265625, 1.267578125, 1.0673828125, -0.072509765625, -0.447265625, 0.55322265625, -1.5869140625, -0.197265625, 0.49560546875, -1.5458984375, -0.2322998046875, 0.06378173828125, -0.222412109375, -0.705078125, -1.845703125, 0.302734375, -1.1279296875, 0.332275390625, -0.9169921875, -0.88330078125, -1.27734375, 0.5703125, 1.970703125, 0.08111572265625, -1.1513671875, -2.236328125, 0.389404296875, 1.7509765625, 1.7802734375, 0.513671875, 1.591796875, -0.304443359375, -0.29296875, 0.52490234375, 1.685546875, -0.43994140625, 0.73828125, 0.67138671875, 1.208984375, 1.0654296875, 0.673828125, -0.165283203125, 0.642578125, -0.96630859375, -0.6103515625, -0.41015625, -0.53662109375, 2.3671875, -0.4609375, 1.361328125, 0.77880859375, -0.07861328125, 0.51611328125, 0.25732421875, 1.4033203125, 0.12103271484375, 0.99755859375, -0.03167724609375, -0.432373046875, 0.84423828125, -0.09710693359375, -1.2734375, 0.342529296875, 0.0645751953125, 0.31884765625, 0.67626953125, 0.1234130859375, 1.2490234375, -0.7666015625, 1.076171875, 0.58447265625, -1.1435546875, -0.2861328125, 0.189453125, -0.64892578125, -0.0882568359375, -0.414794921875, -1.14453125, -0.91796875, 0.7197265625, 0.66552734375, 0.1309814453125, 0.1871337890625, -0.055694580078125, -1.2412109375, -0.58251953125, 0.8134765625, 0.2344970703125, 0.26123046875, -0.26123046875, -0.252197265625, 0.03192138671875, -0.416259765625, 0.5625, -2.23046875, 2.185546875, 0.1978759765625, -0.217041015625, -0.3408203125, 0.2000732421875, -0.1663818359375, -0.6298828125, 0.07928466796875, -0.2462158203125, 1.14453125, -0.79248046875, -0.90869140625, -0.2303466796875, -0.07061767578125, -0.73193359375, 0.3916015625, 1.8798828125, -1.04296875, 1.1767578125, -1.0244140625, 0.00750732421875, -0.473388671875, 0.2176513671875, 1.0244140625, 0.99755859375, -0.6455078125, -0.354736328125, -0.85107421875, -0.5556640625, -0.07855224609375, 0.1900634765625, 1.8232421875, 0.45458984375, 0.28564453125, -0.69970703125, 1.9755859375, 1.484375, 0.17578125, -0.66259765625, 0.86474609375, -1.56640625, -1.5751953125, 0.322265625, -0.662109375, -1.8056640625, 0.8046875, -0.08843994140625, -2.0078125, -1.10546875, -0.71142578125, -1.4150390625, 0.28466796875, -0.97705078125, 0.8857421875, 0.7978515625, 0.37939453125, 1.7568359375, -0.60791015625, -1.26171875, 2.115234375, -0.9716796875, 0.349853515625, -0.391845703125, -0.9619140625, 2.427734375, -0.2705078125, 1.3427734375, -0.876953125, 0.98876953125, 0.8857421875, -0.9775390625, 0.59521484375, 0.11151123046875, -0.73388671875, -2.43359375, -0.61865234375, 0.9892578125, 0.203125, 0.059173583984375, -0.51953125, 1.1259765625, -0.470947265625, 0.2344970703125, -0.69091796875, 0.44287109375, -0.228759765625, -0.91845703125, 1.267578125, -0.33056640625, 0.7880859375, 1.33203125, -1.318359375, -0.6865234375, 0.9873046875, 0.57470703125, -0.377685546875, -0.50830078125, 1.1162109375, -0.260498046875, -0.27197265625, -0.06317138671875, -1.4560546875, -1.216796875, -1.611328125, 0.20361328125, -0.330810546875, 0.55078125, 1.7021484375, -0.53369140625, 1.3310546875, 0.89453125, 0.67236328125, -2.01171875, 2.166015625, -0.1527099609375, 0.82861328125, 0.61083984375, -1.625, 0.254638671875, 0.58349609375, -1.005859375, 0.54931640625, -0.315673828125, 0.138916015625, -1.275390625, -0.7822265625, -0.70703125, 0.2861328125, 0.44970703125, 1.0302734375, 0.1275634765625, 0.59716796875, 0.53271484375, 1.189453125, 0.11767578125, 1.5107421875, 0.8193359375, -0.54931640625, 0.051544189453125, -0.1668701171875, -1.2783203125, -1.015625, -1.1884765625, -0.23828125, 0.67236328125, 0.52392578125, 1.9853515625, 1.197265625, -0.89013671875, -0.79931640625, 1.2861328125, -0.41064453125, 0.39013671875, -0.180419921875, -1.31640625, -0.03912353515625, -0.158203125, 1.5732421875, 0.5302734375, -0.2259521484375, 1.6328125, 0.41455078125, 1.1357421875, -0.8662109375, 0.7958984375, 2.3671875, -0.6494140625, -0.1966552734375, 1.2353515625, 0.82177734375, -1.29296875, -0.64208984375, -0.063720703125, 2.361328125, -0.232421875, -0.490966796875, -0.0509033203125, -0.99853515625, 1.3662109375, 1.0126953125, -0.6728515625, 2.228515625, -0.6669921875, -0.25537109375, -1.0830078125, -0.2354736328125, -0.5185546875, 0.87255859375, -0.25537109375, 1.1943359375, -0.294677734375, 0.27392578125, 1.36328125, -0.8447265625, -0.69189453125, 0.051177978515625, 0.2354736328125, -0.09332275390625, -0.75732421875, 0.12176513671875, 0.50927734375, 1.1845703125, -0.83837890625, -0.73583984375, 0.5439453125, 0.239501953125, -0.52783203125, -0.7314453125, -1.4140625, 2.314453125, -0.3681640625, 0.82373046875, -0.3330078125, -0.07421875, -1.3486328125, -0.432861328125, -1.470703125, -0.445068359375, -0.51416015625, 0.28857421875, -0.91259765625, 0.4404296875, 0.492431640625, -1.40234375, -0.291259765625, 0.418701171875, 0.3701171875, 1.009765625, 1.0185546875, 0.8515625, -1.3466796875, -0.159423828125, 2.376953125, 0.81396484375, 0.268798828125, 0.78564453125, -1.208984375, -0.1883544921875, 0.75537109375, -0.3046875, -0.544921875, -0.302734375, -0.97265625, -0.36328125, -2.138671875, -0.07891845703125, -1.0849609375, -1.73828125, 1.30859375, -0.68896484375, -0.83984375, 0.7138671875, -0.39990234375, -0.68310546875, 0.233154296875, 0.671875, 0.391845703125, 2.9609375, 0.1668701171875, 1.4541015625, 0.8017578125, 0.7568359375, 0.31787109375, -1.267578125, -1.224609375, -0.4765625, -1.2724609375, -0.2108154296875, -0.6259765625, 0.0051727294921875, 1.009765625, 0.08782958984375, 1.2529296875, -1.947265625, -0.035247802734375, -0.90283203125, -0.96142578125, -0.190673828125, 0.356201171875, 0.134765625, 1.103515625, 0.12005615234375, 0.67578125, -1.5908203125, 1.4541015625, -1.0888671875, 0.748046875, 0.52734375, -2.306640625, -0.55224609375, 0.46142578125, 0.193359375, 0.4658203125, -1.4638671875, -1.4921875, 0.079833984375, 0.366943359375, -0.55419921875, 0.12408447265625, -0.250244140625, 0.8603515625, -1.140625, -1.83984375, 0.11309814453125, -0.273193359375, -0.7861328125, -0.406005859375, -1.2578125, 0.1842041015625, -0.1160888671875, 0.8837890625, 0.340087890625, -0.04962158203125, 0.435302734375, -1.1865234375, -2.01171875, -1.837890625, 0.7314453125, -0.45654296875, 0.40185546875, 0.1785888671875, 1.3623046875, 0.3359375, -0.54052734375, 0.316162109375, 1.2001953125, -1.8984375, -0.9287109375, -0.421142578125, -0.233154296875, 1.3037109375, -0.052398681640625, 1.0625, -0.252685546875, -0.19189453125, -0.2464599609375, 0.271484375, 0.11126708984375, 1.7216796875, -2.048828125, -0.66357421875, -0.3974609375, 1.0224609375, 0.513671875, -0.59326171875, 1.791015625, -0.8251953125, -0.80029296875, 0.02764892578125, 0.004550933837890625, -0.73291015625, 1.1513671875, -0.16943359375, -0.935546875, 0.005519866943359375, -0.31005859375, -0.1943359375, 0.66748046875, 0.10150146484375, 0.486572265625, 0.1785888671875, -0.86865234375, 0.3935546875, -1.2080078125, 1.783203125, 0.53271484375, 1.048828125, -0.1529541015625, 0.316650390625, -0.09912109375, 1.90625, 1.208984375, 0.021942138671875, 0.88427734375, 0.9013671875, -0.41455078125, 0.81884765625, -2.140625, -0.9921875, 2.091796875, 1.666015625, -0.56298828125, 0.032257080078125, -0.7001953125, 0.546875, -0.053955078125, -1.1376953125, 0.11529541015625, -0.7060546875, 0.57861328125, -0.34521484375, -0.2344970703125, 0.73388671875, 0.580078125, 0.484619140625, 0.81103515625, -0.8310546875, -0.1416015625, 1.4560546875, -1.451171875, 0.52294921875, -0.8056640625, -1.2998046875, -0.62939453125, -0.5224609375, 1.2255859375, -1.9677734375, -0.415283203125, -1.6875, -0.06658935546875, 0.41552734375, 0.033935546875, 1.3876953125, 1.1787109375, -1.275390625, -0.771484375, 1.23828125, 1.072265625, 1.6044921875, 0.1044921875, -0.1243896484375, -1.1630859375, 2.796875, 0.2410888671875, 1.1259765625, 0.283935546875, -0.3701171875, 0.98388671875, 0.11944580078125, -0.32373046875, 0.5927734375, 1.0087890625, -0.1707763671875, -1.09765625, -2.791015625, -1.453125, 0.0618896484375, 0.91259765625, -0.205322265625, -0.11309814453125, 0.111572265625, 0.293701171875, 0.68994140625, 0.77392578125, -0.7158203125, 0.187744140625, 1.5234375, -0.408935546875, -0.77001953125, -1.7490234375, -1.7041015625, 0.87158203125, 2.150390625, -0.48779296875, -1.1865234375, -0.26416015625, 0.403564453125, 0.4453125, 0.07830810546875, -0.57958984375, -0.0223236083984375, 0.30810546875, -0.5595703125, 0.436279296875, -0.57080078125, 0.4267578125, -0.385009765625, -1.6435546875, -0.583984375, -1.4931640625, -0.34375, 0.91650390625, 0.7138671875, 0.00916290283203125, 0.322021484375, 0.22314453125, 0.1666259765625, -1.19140625, -0.68798828125, -0.177490234375, 1.455078125, -0.66650390625, -0.0111083984375, -1.470703125, -0.59228515625, 0.05230712890625, 1.0107421875, -0.880859375, 1.74609375, -0.6845703125, 0.1942138671875, 1.947265625, -0.63720703125, 1.005859375, 0.9736328125, 1.818359375, 1.6298828125, 0.65625, 1.220703125, 1.162109375, -0.57373046875, -0.09765625, 1.0703125, -1.1943359375, -2.380859375, -0.578125, -0.5673828125, 1.107421875, 0.1783447265625, -0.998046875, -1.0693359375, -0.611328125, -0.5341796875, -0.62841796875, -1.30859375, -1.5234375, -1.6611328125, 0.5693359375, -1.396484375, -0.343017578125, 0.460205078125, -0.57958984375, -0.8720703125, 0.82421875, -0.8115234375, -0.8388671875, 0.26416015625, -0.484619140625, -0.458984375, 0.6728515625, 0.78125, 0.5400390625, -0.0155487060546875, -0.05322265625, 0.006443023681640625, 1.4072265625, -0.386474609375, 0.611328125, 0.1612548828125, -1.482421875, -1.5576171875, -0.6025390625, 1.712890625, -1.013671875, -0.623046875, -0.03045654296875, 0.73388671875, 0.1446533203125, 0.54541015625, 0.93359375, -1.54296875, 1.087890625, -0.8720703125, 1.70703125, -1.6650390625, -1.1015625, -0.3271484375, 0.35107421875, 0.6435546875, -0.402587890625, 0.009765625, -0.1339111328125, -1.451171875, 1.177734375, -0.440673828125, -0.71142578125, -0.51171875, -0.6513671875, -0.8935546875, 1.87890625, 2.34765625, -0.9482421875, 1.4755859375, -2.322265625, -0.6083984375, -0.095703125, 0.81396484375, 0.73046875, 0.650390625, -0.2379150390625, 1.1982421875, 0.3212890625, 0.39892578125, 0.495361328125, -0.3740234375, -0.41259765625, 0.55078125, 0.26904296875, -1.3193359375, 1.5859375, 0.880859375, -0.85693359375, -0.0755615234375, -1.1962890625, -0.286865234375, 1.171875, -1.4326171875, 0.287353515625, -1.3876953125, -0.10589599609375, 0.64453125, 0.8974609375, 0.446533203125, 0.99169921875, -1.0048828125, 1.6552734375, -0.6943359375, 0.2763671875, -0.8427734375, 1.220703125, 1.1396484375, -0.359619140625, -0.2406005859375, -0.56298828125, -3.197265625, -0.86328125, 2.1640625, -1.447265625, -0.493408203125, 0.2069091796875, -0.54638671875, -1.1064453125, -1.251953125, -0.1295166015625, -0.92041015625, -0.468994140625, -2.59765625, -0.76708984375, 0.490478515625, -0.477783203125, 0.63671875, -0.98046875, -2.083984375, -0.802734375, -0.0144805908203125, -0.073486328125, -0.03228759765625, 1.466796875, 0.51171875, -0.2220458984375, 2.259765625, 1.2939453125, 0.4482421875, -0.6376953125, -0.250244140625, 0.0148468017578125, -3.009765625, -0.155029296875, -0.8154296875, -0.0648193359375, -0.469970703125, -1.31640625, -0.68994140625, 1.71875, -0.409912109375, -0.56640625, -0.0555419921875, -0.9443359375, -1.4091796875, -1.0634765625, -1.572265625, 0.1259765625, -0.68212890625, -1.9462890625, 2.251953125, 1.8876953125, -0.13037109375, -0.98291015625, 0.3564453125, 0.357666015625, -0.1605224609375, 1.0673828125, -0.037017822265625, 1.005859375, 0.51171875, 3.470703125, 1.1298828125, 0.568359375, 1.0263671875, -1.8369140625, 0.83203125, 1.76953125, 1.1884765625, -1.138671875, -1.0517578125, -0.38720703125, -0.5205078125, -0.479248046875, 3.646484375, 0.95458984375, 0.7236328125, -0.30224609375, -0.9619140625, 0.163818359375, 0.430419921875, 1.3125, -0.81201171875, -0.1480712890625, -1.357421875, 0.0526123046875, -0.87744140625, 2.052734375, 0.2401123046875, -1.0029296875, -1.201171875, 0.1134033203125, 0.669921875, 2.16796875, -0.39990234375, -0.438232421875, 0.1572265625, 0.90625, 2.646484375, 1.197265625, -0.09210205078125, -0.70068359375, 0.30712890625, 1.076171875, 0.546875, -0.31298828125, -1.30078125, 1.0810546875, 0.1439208984375, -0.1849365234375, -0.6953125, 2.056640625, 1.5576171875, 0.6474609375, -1.9658203125, -1.46484375, -2.04296875, -1.1162109375, 0.86962890625, 0.91552734375, -0.60986328125, 1.3271484375, -0.513671875, 1.8173828125, -0.0083770751953125, 1.0478515625, -0.10931396484375, -1.205078125, -1.8037109375, -1.3994140625, 1.2119140625, 0.095703125, -0.3154296875, -0.1436767578125, -2.458984375, 0.86572265625, 0.58935546875, -0.365966796875, 1.76171875, -2.189453125, 0.93310546875, -0.8515625, -1.8515625, 1.1259765625, -0.44775390625, -1.1201171875, 0.8671875, -0.362548828125, 0.2305908203125, -0.48486328125, -0.5634765625, -0.31494140625, -0.57373046875, -0.1240234375, -0.0528564453125, 1.109375, 1.0048828125, 0.462646484375, 0.802734375, -0.78173828125, -0.1575927734375, 0.259765625, 1.6923828125, -0.80810546875, 0.76416015625, 1.5478515625, -0.9853515625, -0.8828125, -0.1904296875, 0.2113037109375, 0.728515625, 0.45849609375, 1.662109375, -0.6142578125, -1.298828125, -0.04437255859375, -1.19921875, -0.830078125, 1.0712890625, -1.51953125, -0.84326171875, -0.9326171875, 0.277099609375, 1.1357421875, 0.81982421875, -0.8603515625, -0.5517578125, -0.6123046875, 2.10546875, 0.273193359375, 0.50537109375, 0.275634765625, -0.050872802734375, 0.953125, -0.1290283203125, 1.11328125, 0.5224609375, -1.099609375, -1.6376953125, -1.1962890625, 0.939453125, -1.904296875, 1.3603515625, 0.748046875, 0.90771484375, 0.08184814453125, 0.2666015625, -0.2381591796875, -0.1678466796875, -0.76611328125, 0.2607421875, -1.2373046875, -0.76513671875, -0.00662994384765625, -0.318115234375, 1.10546875, 0.1439208984375, 0.81591796875, -0.053253173828125, 2.369140625, 0.238525390625, -0.250244140625, 0.59912109375, 0.75927734375, -0.250244140625, 1.5380859375, 1.498046875, -1.0126953125, -0.77685546875, 0.58740234375, 0.52685546875, -0.1097412109375, 0.465576171875, -0.3818359375, -0.11090087890625, 0.3349609375, 1.36328125, 0.322265625, -0.416015625, -0.269287109375, -0.0928955078125, -0.03411865234375, -0.48095703125, 0.90478515625, 1.4169921875, -0.8955078125, -0.469970703125, 0.42138671875, 0.439208984375, -0.5732421875, -1.53515625, 0.33251953125, 0.319580078125, 0.441162109375, -2.515625, 1.4990234375, -0.783203125, 2.271484375, 0.143798828125, -0.67724609375, 0.499755859375, 0.68115234375, 1.306640625, -0.55810546875, -2.740234375, 0.477783203125, -1.0185546875, 0.251220703125, -0.841796875, -0.6796875, -0.837890625, -0.302490234375, 0.0889892578125, 0.80419921875, 0.41796875, -1.1875, -1.13671875, 1.1572265625, -0.9482421875, 0.77978515625, 1.177734375, -1.7392578125, -0.44140625, 1.6669921875, 0.09912109375, -0.11077880859375, -0.293701171875, 0.5693359375, 0.195068359375, 0.086669921875, -0.5078125, -0.4794921875, 0.912109375, 0.93798828125, -1.0654296875, -0.77490234375, 1.8271484375, 0.646484375, 0.6494140625, 0.3583984375, 0.39501953125, 0.75, 1.1640625, -0.7626953125, -0.59228515625, -0.08917236328125, 0.06170654296875, -2.107421875, -0.062744140625, 1.2412109375, -0.1370849609375, -1.12109375, -0.88134765625, -0.87451171875, -0.1900634765625, 0.221923828125, 0.1676025390625, -1.0234375, -0.05230712890625, 0.72216796875, 1.5703125, -1.14453125, -0.6181640625, 1.109375, -0.79833984375, -0.165771484375, -0.10284423828125, 0.06884765625, -0.978515625, -1.5947265625, 0.0548095703125, -0.4208984375, 0.0692138671875, 1.53515625, -1.765625, -0.90576171875, -1.091796875, -0.52685546875, -0.0831298828125, -0.88671875, -1.296875, 0.62744140625, 0.382568359375, 0.9033203125, 0.93212890625, 0.78271484375, -0.440673828125, -0.213623046875, -0.97509765625, -1.4296875, -0.60595703125, 1.7353515625, 0.65966796875, 1.58984375, 0.0077667236328125, 0.69384765625, -0.70947265625, -1.115234375, 0.95556640625, -1.216796875, -0.43505859375, 0.2200927734375, -0.249267578125, 0.438232421875, 1.3720703125, -1.66015625, 0.40185546875, -2.06640625, 1.3603515625, 1.505859375, 0.320068359375, 0.1964111328125, -0.8974609375, 2.369140625, 0.1412353515625, -0.34326171875, 1.3203125, -0.485107421875, 0.841796875, -0.038177490234375, -0.109375, -0.186767578125, -0.025543212890625, -1.6171875, 0.52783203125, -0.6923828125, -0.95263671875, 0.52880859375, 0.2474365234375, -0.97607421875, 0.1597900390625, -0.5546875, -0.96435546875, 1.6455078125, 0.654296875, 0.86181640625, -0.94384765625, 1.3671875, 1.6513671875, -1.205078125, 1.7197265625, 0.428955078125, -0.91845703125, 1.5703125, -1.154296875, -1.125, 0.841796875, -0.0161895751953125, 0.9619140625, -0.892578125, -1.06640625, -1.41796875, -0.72998046875, -1.0703125, 0.201416015625, 1.189453125, 1.3203125, -0.1171875, -1.0654296875, 0.69921875, 0.324462890625, -2.15234375, 1.34375, 0.470458984375, -1.330078125, -0.1158447265625, -1.236328125, 0.6181640625, -0.7861328125, 0.7802734375, -0.44775390625, 0.2099609375, 0.08575439453125, -1.8857421875, -0.050018310546875, 0.332763671875, -2.1171875, 0.32080078125, -0.4599609375, 0.6904296875, -1.208984375, 0.348388671875, 0.182373046875, -0.65771484375, 0.552734375, -0.68310546875, -0.60546875, 1.73828125, 0.072509765625, 1.927734375, -1.47265625, 0.70751953125, -0.11968994140625, -0.6689453125, 1.0478515625, -0.094970703125, -0.58642578125, 0.6845703125, -1.154296875, -0.9365234375, -0.3623046875, 1.509765625, 0.05499267578125, 0.6513671875, 0.54931640625, 0.7822265625, -0.209228515625, -0.113037109375, -0.86669921875, 2.169921875, 0.2432861328125, 0.4482421875, 1.970703125, -0.214599609375, -0.3173828125, 0.56640625, 1.0205078125, -1.0869140625, 0.1370849609375, 0.25732421875, -1.0712890625, 0.265380859375, 0.91259765625, -0.6904296875, 0.274658203125, 1.708984375, -0.83837890625, -0.253662109375, -0.1455078125, 2.482421875, -0.8056640625, 1.8271484375, 0.4326171875, 0.365478515625, -1.140625, 1.4306640625, 0.4345703125, 0.71875, -0.272216796875, -1.2275390625, 1.953125, 0.498046875, 0.301513671875, 0.73828125, 1.763671875, -0.68115234375, 1.5146484375, 1.0732421875, -0.7236328125, 0.4150390625, 0.09820556640625, 2.619140625, 1.9482421875, -0.448486328125, 0.55517578125, 0.85498046875, -0.64404296875, 0.276611328125, -0.509765625, 0.17822265625, 1.03125, 0.27001953125, 1.5556640625, -0.36962890625, -0.0297088623046875, 0.92236328125, 0.1651611328125, 0.282958984375, 0.73486328125, 0.875, -0.7646484375, 0.96533203125, 0.219970703125, -0.91015625, 0.251220703125, 1.0458984375, -0.81494140625, 1.6884765625, -0.1396484375, 1.96484375, 0.28955078125, 0.186279296875, 3.03515625, -0.345947265625, 0.67822265625, 0.270751953125, -1.0732421875, -1.04296875, -0.431640625, 0.467041015625, 0.93505859375, 2.880859375, 0.8408203125, -0.87255859375, -0.19677734375, 1.462890625, 0.28515625, -0.42138671875, -0.462890625, -1.98046875, -0.447998046875, -0.069091796875, -1.255859375, -1.056640625, 0.0653076171875, 1.1279296875, -1.5341796875, 0.75927734375, -0.52490234375, 1.5146484375, 1.1298828125, -0.1575927734375, 0.6240234375, 0.26220703125, -0.74365234375, -0.14697265625, -0.426513671875, -1.82421875, 1.4599609375, 0.9619140625, 0.1695556640625, -0.67724609375, -0.44677734375, -0.52978515625, -1.0224609375, 0.0281219482421875, -0.0948486328125, 0.492919921875, -1.1748046875, -1.8818359375, -0.4365234375, -1.111328125, -1.3857421875, -0.80859375, -2.40625, 0.7333984375, -0.151611328125, -0.10064697265625, 0.31982421875, 0.6904296875, -0.75927734375, -0.91845703125, 0.170654296875, -0.1749267578125, -0.045806884765625, -0.0845947265625, 0.0255126953125, 1.103515625, 0.9541015625, -0.278564453125, 1.3857421875, 1.1748046875, -0.1229248046875, -2.66015625, -1.638671875, 0.53857421875, 0.28271484375, 0.54150390625, -0.52392578125, -0.254150390625, -1.1650390625, 0.15380859375, 1.052734375, 0.490234375, 1.109375, 1.609375, 0.5107421875, -0.440185546875, -0.1116943359375, -0.8095703125, 1.025390625, -0.6328125, -0.916015625, -0.80615234375, -1.9931640625, 1.40234375, 2.255859375, 0.35888671875, -1.0341796875, -0.6943359375, 0.387451171875, -0.22509765625, 0.2257080078125, -0.90087890625, -2.40625, 0.880859375, 0.487548828125, -1.310546875, -0.5126953125, 0.47314453125, 1.10546875, -0.130615234375, -0.2176513671875, -1.0634765625, 0.748046875, 1.8623046875, 1.953125, 0.62890625, -0.405029296875, -0.2076416015625, -0.27490234375, -1.0009765625, -0.410400390625, 1.484375, -0.7900390625, 0.2783203125, -0.93701171875, -0.4384765625, 0.50830078125, 0.229736328125, 0.7607421875, 0.59619140625, -1.7666015625, 0.36181640625, 0.73583984375, -0.76416015625, 0.70947265625, -0.48388671875, 0.78271484375, 0.8662109375, -0.398193359375, -0.032989501953125, -1.203125, -0.078369140625, 0.1461181640625, 0.2137451171875, -0.46142578125, 0.34375, -2.943359375, -0.966796875, -0.44384765625, 1.46484375, 2.015625, -0.8759765625, -1.4775390625, 0.935546875, 0.025054931640625, 0.85693359375, -1.83984375, -0.7666015625, -0.484130859375, -1.2880859375, -0.89697265625, 0.07513427734375, -1.0576171875, 0.3154296875, -0.9697265625, -0.478271484375, 2.392578125, 0.048492431640625, -1.1728515625, 0.77001953125, -0.2265625, 1.953125, -1.7216796875, -1.1298828125, -1.8916015625, -0.86328125, 0.061737060546875, 1.2412109375, 0.261962890625, 0.66015625, 0.053955078125, -1.271484375, 0.447998046875, -0.99560546875, 0.91748046875, 0.92529296875, -0.1163330078125, -0.751953125, 0.53173828125, -0.365966796875, -0.99267578125, -0.1356201171875, -2.791015625, 1.9677734375, 0.060150146484375, -1.173828125, -0.437255859375, 0.012725830078125, 0.79248046875, 0.5419921875, 0.98828125, 0.0673828125, -0.01007843017578125, 1.38671875, 0.6435546875, -0.689453125, 1.306640625, 0.173095703125, 0.51904296875, 1.1826171875, -0.31591796875, 0.83740234375, -0.99462890625, -0.72314453125, 0.0908203125, -0.8408203125, 2.4921875, 0.416015625, 0.12841796875, 1.298828125, 0.263671875, -1.470703125, -0.982421875, 0.25927734375, 0.20166015625, -0.00649261474609375, -1.4306640625, 0.321533203125, -0.404052734375, -1.3212890625, 0.5205078125, 0.31005859375, 0.0283355712890625, 0.43798828125, 0.05865478515625, 0.671875, 0.53466796875, -0.3134765625, -0.7861328125, -0.338623046875, -0.0587158203125, -0.08380126953125, 0.47705078125, -0.658203125, -0.00881195068359375, 0.035430908203125, 1.8134765625, -0.387451171875, -0.109130859375, 0.69970703125, -0.12054443359375, 0.736328125, -0.25732421875, 1.4296875, 0.157470703125, -0.14794921875, -0.794921875, -0.5048828125, 1.7880859375, 0.6123046875, 2.115234375, -0.1502685546875, 1.103515625, -1.3193359375, 0.315673828125, 0.2110595703125, -0.37109375, 0.437744140625, 0.52001953125, 1.119140625, -1.4541015625, -2.154296875, -1.4619140625, -0.1009521484375, -1.4150390625, -1.076171875, 0.140625, -0.908203125, 0.3203125, 0.08746337890625, 0.55126953125, 0.875, -1.4609375, -0.392822265625, 0.5263671875, 0.385009765625, 1.1826171875, 1.509765625, 0.08343505859375, 0.367431640625, 0.31689453125, -0.4033203125, 0.6923828125, 0.8154296875, 0.52734375, 0.06903076171875, -0.78271484375, 0.4541015625, 0.346435546875, 0.2548828125, -1.3505859375, -0.90478515625, -0.462158203125, 1.8916015625, -0.56787109375, 1.2060546875, -1.638671875, 1.169921875, -1.7294921875, -0.433349609375, 1.833984375, -0.78173828125, 1.8232421875, -0.0261383056640625, 0.62548828125, -0.28271484375, 0.1085205078125, 0.337890625, 0.1221923828125, -1.3203125, -0.0268707275390625, 1.517578125, -0.7216796875, 1.9130859375, 1.7685546875, 1.951171875, -0.90673828125, 0.45556640625, -1.6455078125, -0.047027587890625, 0.16748046875, -0.58740234375, -2.05859375, -0.4267578125, 0.91748046875, 3.685546875, -1.3017578125, -0.72265625, -0.560546875, 0.6455078125, -0.18212890625, 0.666015625, 0.75341796875, 0.350341796875, 0.87109375, 0.416259765625, 0.27783203125, 0.65380859375, 0.607421875, 0.1436767578125, 0.28173828125, 0.50244140625, 0.68896484375, 0.92919921875, -1.9111328125, 1.3388671875, 0.08575439453125, -0.390380859375, -0.0831298828125, -1.2998046875, 0.80224609375, -0.056732177734375, -0.1783447265625, 1.72265625, -0.2462158203125, -0.18896484375, 0.5380859375, 0.3798828125, 1.1962890625, -0.64892578125, 1.7451171875, -0.53173828125, -0.68994140625, 1.65234375, 0.74267578125, -0.242919921875, 1.26171875, -2.5234375, 0.56005859375, -0.4208984375, 0.352294921875, 0.74853515625, -2.033203125, -0.0787353515625, 2.0390625, -0.0579833984375, -0.2034912109375, -0.29150390625, -0.6953125, 2.52734375, -0.1287841796875, 1.68359375, -0.275634765625, 0.97607421875, -0.88916015625, 0.159912109375, -1.65234375, 0.0802001953125, 0.1661376953125, -1.1005859375, 0.7412109375, -0.300048828125, 2.294921875, -0.50048828125, 0.43701171875, 0.367919921875, 0.67431640625, -1.642578125, -0.463623046875, 0.9326171875, 0.191162109375, -0.66357421875, 0.0081329345703125, -0.81396484375, -0.5341796875, 0.76220703125, -0.55712890625, 0.489013671875, -0.458984375, 0.6142578125, -0.60205078125, -0.60302734375, 0.2958984375, 0.7138671875, 0.240234375, 0.134521484375, 0.271240234375, 0.724609375, 0.1319580078125, 0.712890625, 1.123046875, -0.4052734375, 0.9501953125, -2.03125, 0.5126953125, -0.10174560546875, 0.9453125, -0.28076171875, -0.8515625, -0.189453125, -1.2783203125, 0.69140625, -0.6650390625, -1.201171875, -0.5263671875, -0.78369140625, 1.49609375, -0.8623046875, -0.489501953125, 0.452880859375, -0.2998046875, -0.81396484375, -1.490234375, -0.75830078125, -0.05145263671875, 0.4267578125, -0.9833984375, 2.08984375, -0.6015625, -0.50634765625, 0.9130859375, 0.7724609375, 0.4658203125, 0.303466796875, -0.8701171875, -0.0938720703125, -1.455078125, 1.0859375, -0.07550048828125, 1.1591796875, 0.87158203125, 0.266845703125, -0.7001953125, -0.87744140625, 0.484619140625, 0.082275390625, -0.2578125, -0.19677734375, -0.78564453125, -0.322265625, -0.007465362548828125, -0.99267578125, -0.06939697265625, -1.408203125, -0.2210693359375, 1.4970703125, -0.2401123046875, 0.37158203125, 0.028106689453125, -0.419921875, -1.484375, 0.64453125, -2.03515625, -1.8203125, -0.71044921875, 0.728515625, -0.458251953125, 1.1962890625, 0.66064453125, -0.381591796875, 1.4814453125, 0.6103515625, -1.30859375, -0.3154296875, 0.69482421875, -0.7880859375, -1.287109375, 1.8271484375, -0.6220703125, -0.2288818359375, -0.2305908203125, 0.65380859375, -0.458740234375, 0.93212890625, -0.30419921875, 0.2293701171875, -1.740234375, -0.451171875, 1.5302734375, -1.5029296875, -0.8720703125, -1.2216796875, 0.73974609375, -0.82080078125, -0.00440216064453125, -1.5029296875, -0.99560546875, 0.90576171875, 0.050750732421875, -0.023468017578125, 0.2177734375, -0.428955078125, -2.0234375, -0.41748046875, 0.53857421875, 0.3515625, -0.51806640625, 0.57763671875, -1.1806640625, 0.1778564453125, -0.490478515625, -0.89697265625, 0.80419921875, 0.0050201416015625, 0.974609375, -0.33935546875, 0.5068359375, 1.2998046875, -1.158203125, -0.99365234375, -0.466796875, 0.60791015625, -1.4052734375, 1.234375, -0.35009765625, -0.701171875, -1.2373046875, 0.06427001953125, -1.3681640625, -0.88330078125, -1.2060546875, 0.0249481201171875, -1.5732421875, 0.77734375, -2.205078125, -2.3046875, -0.61474609375, 1.75, 0.1676025390625, 0.2509765625, 2.591796875, -1.3603515625, 0.576171875, 2.13671875, 1.10546875, 0.1661376953125, 0.8818359375, -0.56884765625, -0.10845947265625, 0.59228515625, 1.755859375, -0.55419921875, -0.5205078125, -1.0888671875, 0.0869140625, 0.427001953125, 1.671875, -0.13525390625, -0.350341796875, -0.35791015625, 0.740234375, -1.4248046875, -0.83447265625, -0.6650390625, -1.337890625, -1.1884765625, 0.6220703125, -1.6123046875, 0.98193359375, -0.267333984375, -2.08984375, -0.331787109375, 0.89404296875, -0.1220703125, 0.1630859375, 0.2279052734375, 0.472412109375, 0.306884765625, -0.30419921875, -1.353515625, 0.60546875, 1.482421875, 0.68994140625, 0.62744140625, 1.2958984375, 0.6943359375, -0.12371826171875, -0.336181640625, -0.525390625, -1.2587890625, 0.98046875, 0.3212890625, 0.4873046875, 1.4013671875, 1.3291015625, -0.7958984375, 1.2587890625, 0.607421875, -1.8828125, -1.76171875, 0.047760009765625, -0.87744140625, -1.51171875, 0.56689453125, 0.77587890625, 2.50390625, 1.5947265625, -0.78271484375, 0.56884765625, 0.5791015625, 0.16943359375, -1.037109375, -0.412353515625, -1.216796875, -0.6650390625, 0.146240234375, -0.1815185546875, 0.5185546875, 0.0830078125, -0.90625, 0.1279296875, -1.7255859375, 0.353271484375, -0.01496124267578125, 0.158447265625, 0.98779296875, 0.728515625, -0.05072021484375, -0.8017578125, -1.212890625, -0.200439453125, 1.2060546875, -0.978515625, 1.4443359375, 0.8369140625, -1.634765625, 0.6494140625, 1.2197265625, -0.8671875, -0.67822265625, 1.037109375, -0.0791015625, -0.08184814453125, -0.10858154296875, 0.181396484375, 0.0599365234375, -0.330322265625, -0.6953125, 1.619140625, -0.861328125, -1.0927734375, -1.0068359375, 1.2548828125, -0.3125, -2.45703125, 0.45361328125, -2.126953125, 0.91357421875, 0.07904052734375, -0.7998046875, 1.8720703125, 1.220703125, -0.8740234375, -0.521484375, -0.70947265625, -1.6240234375, -0.296875, -1.345703125, 0.06048583984375, -0.6298828125, 1.0791015625, 0.44189453125, -0.455810546875, -0.92529296875, 1.3388671875, 1.1376953125, -1.505859375, -0.43798828125, -0.0880126953125, -0.96240234375, -0.39501953125, -1.396484375, -0.045654296875, -1.177734375, -0.08587646484375, 0.71240234375, 0.529296875, -0.320068359375, -0.64697265625, -2.044921875, 0.494384765625, -1.4228515625, -1.015625, 1.30078125, 0.7138671875, 0.58154296875, 0.80908203125, -1.806640625, -0.376220703125, 0.34423828125, 2.13671875, 0.1551513671875, -0.328857421875, 2.0859375, 0.2178955078125, 0.587890625, -2.041015625, -0.70751953125, -1.21875, -0.406005859375, 0.1663818359375, 0.5, 0.8173828125, 1.79296875, -0.845703125, -0.712890625, -0.26513671875, -0.0950927734375, 1.1396484375, -0.2115478515625, 1.4921875, -0.62451171875, -0.68115234375, -0.0294342041015625, -0.9228515625, -1.693359375, -0.830078125, 0.546875, 0.4375, 1.4189453125, 1.3642578125, 0.55810546875, 0.71923828125, -1.9912109375, -0.76220703125, 2.099609375, -0.904296875, 0.6640625, 0.8251953125, -1.478515625, -0.485595703125, 1.3125, 0.6728515625, -0.2066650390625, 0.12286376953125, 0.6748046875, -0.5126953125, 1.306640625, 1.234375, -0.0677490234375, 0.04083251953125, -0.207763671875, -0.75537109375, 0.280517578125, -1.25, 0.5126953125, 0.76123046875, -0.43017578125, -1.2822265625, 0.43310546875, 0.07574462890625, 2.89453125, -0.498046875, -0.92822265625, 0.248779296875, 0.358154296875, 0.67529296875, 0.173583984375, 1.7626953125, -0.357421875, 0.35107421875, 0.91162109375, -0.4072265625, 0.1292724609375, -1.591796875, -2.056640625, 0.0809326171875, -0.9287109375, -2.619140625, 0.78515625, 0.0020694732666015625, -0.7783203125, -0.489990234375, 0.09228515625, -1.107421875, 0.70654296875, 0.01424407958984375, 0.352294921875, 0.1309814453125, 0.1434326171875, 0.9970703125, -2.716796875, -0.078369140625, -0.501953125, -0.5458984375, 0.042694091796875, -0.9638671875, 2.05078125, 0.822265625, -0.712890625, 0.218505859375, -1.1865234375, -1.6923828125, 0.335693359375, 0.51220703125, -1.37109375, 1.2578125, 1.6494140625, 0.6328125, -0.289794921875, 0.126953125, -0.368408203125, 1.0625, -0.484130859375, 1.01953125, 0.97607421875, -0.7958984375, 0.84033203125, 0.225341796875, -0.90087890625, -0.27001953125, -2.216796875, 0.99267578125, 0.278076171875, 0.0009832382202148438, -1.224609375, -0.169189453125, -0.298095703125, -0.822265625, 0.1112060546875, -1.0830078125, 1.25, -1.19140625, 0.246826171875, 0.97216796875, 1.7451171875, -1.3017578125, 2.115234375, 2.123046875, 0.60888671875, 0.180419921875, -2.158203125, -0.798828125, 0.37255859375, -0.44970703125, -0.1478271484375, 0.13623046875, 0.98095703125, -1.3681640625, 0.9208984375, -2.2421875, 0.88525390625, 0.270751953125, 0.28662109375, 0.12152099609375, -0.405029296875, -1.54296875, 1.0791015625, -1.4853515625, -0.52392578125, 0.028656005859375, 0.053619384765625, -0.256591796875, -0.2880859375, -0.197265625, -0.1719970703125, -0.105224609375, -1.66015625, 0.53662109375, -0.97265625, 0.08172607421875, -1.2646484375, 1.5771484375, 0.73828125, 1.1015625, -1.337890625, -0.06988525390625, -1.798828125, 0.53466796875, -0.9443359375, -0.50537109375, -0.72119140625, 0.263427734375, 1.076171875, 0.51611328125, 0.022308349609375, 1.6337890625, -1.4638671875, 0.1077880859375, -0.81884765625, -1.244140625, -0.748046875, 0.186279296875, -0.48681640625, 0.45556640625, -0.5390625, 1.53125, -0.47021484375, -1.26953125, 2.080078125, 0.68994140625, -1.734375, 1.2001953125, 0.348876953125, 0.1119384765625, 0.61669921875, -0.1522216796875, -0.259033203125, -0.8330078125, -1.5732421875, -1.599609375, 0.50341796875, 0.66259765625, -2.587890625, -0.230712890625, 0.130126953125, -0.53662109375, 1.0341796875, -1.8544921875, -1.154296875, 2.16796875, -0.055938720703125, -0.509765625, -0.035247802734375, 2.40234375, 0.0258331298828125, 0.71484375, -1.6201171875, -0.03662109375, -1.1025390625, 2.625, -1.3017578125, -0.10076904296875, 1.65625, 1.3212890625, -1.00390625, 1.099609375, 2.658203125, -0.333984375, 0.84375, 0.43212890625, -0.34130859375, -0.8984375, 0.40478515625, 1.5078125, 0.92626953125, 0.7236328125, 0.125244140625, 0.08319091796875, 0.14111328125, 1.0751953125, 0.9931640625, 0.10174560546875, 1.6708984375, 1.353515625, -0.96142578125, -0.16357421875, -0.58154296875, -0.61083984375, 0.265869140625, 1.1767578125, 1.35546875, -0.1531982421875, -0.88330078125, -0.7099609375, 0.68212890625, 1.099609375, 0.59619140625, 1.2265625, 0.0865478515625, -1.7421875, -0.2239990234375, -0.314697265625, -0.96875, 0.274658203125, -0.51513671875, -1.06640625, 0.10711669921875, 0.26318359375, 1.5078125, -0.290283203125, 0.1373291015625, -2.427734375, 0.69775390625, -1.5400390625, 0.92724609375, 0.71630859375, 0.3076171875, -0.499267578125, 0.0684814453125, -0.6845703125, -0.66845703125, -0.82421875, -0.06561279296875, -0.11688232421875, -0.73291015625, -0.370849609375, -0.97509765625, -0.75244140625, 1.7724609375, 0.71240234375, -1.3642578125, 0.4599609375, 0.10003662109375, -0.06378173828125, -0.97900390625, -1.662109375, 0.828125, -1.1298828125, -0.2396240234375, -1.220703125, -1.240234375, 0.88232421875, -1.1376953125, -1.353515625, 0.1600341796875, 0.58544921875, 0.708984375, 0.46435546875, -1.4453125, -0.953125, 0.39697265625, -2.2578125, 0.64697265625, 1.6064453125, -2.01171875, -0.54150390625, -0.9482421875, -0.2198486328125, 0.218017578125, -0.2060546875, -1.84765625, 0.7568359375, -0.4970703125, 0.29736328125, -0.031829833984375, 1.017578125, -0.60986328125, 0.765625, 0.44140625, 0.9990234375, -0.57080078125, -1.529296875, -0.37109375, 0.66845703125, -0.143310546875, 1.666015625, 0.460693359375, 0.56640625, -0.71875, -0.2496337890625, 0.86083984375, 0.9609375, -0.253662109375, -0.266845703125, 1.5947265625, 0.144287109375, -0.5234375, -0.77587890625, 0.8330078125, 1.435546875, 0.1016845703125, -0.43603515625, 1.1064453125, 1.60546875, -0.55615234375, 0.304931640625, -0.5859375, -0.146240234375, -0.051300048828125, -0.56201171875, -1.171875, 1.169921875, 0.947265625, -0.468017578125, -0.06585693359375, 0.1116943359375, 0.53759765625, 0.0347900390625, -0.4072265625, 0.88037109375, -0.5966796875, 0.31396484375, -1.0185546875, 1.5380859375, -0.78466796875, 0.3154296875, -0.1593017578125, 0.81787109375, 0.2158203125, -1.1474609375, 0.60302734375, -1.724609375, -0.04461669921875, 0.818359375, 1.4326171875, -1.1298828125, -0.08447265625, -0.12237548828125, 0.68896484375, -0.235595703125, 0.240478515625, -0.48193359375, 1.1513671875, 0.37939453125, -1.220703125, -0.4873046875, 0.6552734375, 0.35205078125, 0.7080078125, -2.05859375, -0.0208587646484375, 0.8359375, 0.7724609375, -2.36328125, -0.389404296875, 0.37353515625, -0.72900390625, 1.6484375, -1.10546875, -0.450927734375, 0.23388671875, -1.10546875, 1.208984375, -0.3740234375, 0.5908203125, -0.452880859375, 0.2080078125, 0.465576171875, -0.0626220703125, -1.5400390625, 0.62451171875, 0.90087890625, 0.212646484375, 1.318359375, -0.95703125, -0.22216796875, -0.136474609375, 1.099609375, 0.91357421875, 0.2115478515625, -0.9609375, -0.3037109375, 0.171630859375, 0.8388671875, 0.357421875, -0.5126953125, 0.99658203125, 1.4365234375, 0.21435546875, -0.626953125, 0.444580078125, -1.0654296875, 1.7490234375, -0.176513671875, 0.146240234375, 0.8603515625, -0.9375, 1.013671875, 1.0615234375, -1.201171875, -0.52197265625, 0.3271484375, 0.1715087890625, -0.32666015625, 0.58837890625, 1.0078125, -0.4462890625, 0.68603515625, 0.951171875, 1.2646484375, 0.318359375, -0.541015625, 0.74169921875, 0.521484375, 1.7138671875, -0.09149169921875, 0.55126953125, -0.327880859375, 2.9609375, -0.299072265625, -0.6669921875, -0.1251220703125, 0.1376953125, 1.193359375, 0.0654296875, 0.7880859375, 0.09814453125, -0.62158203125, 1.2119140625, 0.23876953125, -1.0400390625, 0.7353515625, -0.258544921875, -1.8115234375, -1.1318359375, 1.2255859375, 0.53173828125, -0.62353515625, -1.1240234375, 0.1510009765625, -0.60107421875, -0.2076416015625, 0.5263671875, -0.619140625, -1.478515625, 0.5888671875, 0.308837890625, 0.9853515625, 0.10791015625, -1.974609375, -0.060943603515625, 1.2744140625, -0.48828125, -0.70263671875, -0.61669921875, 0.01434326171875, -2.1484375, 0.470458984375, 0.1885986328125, -0.08392333984375, -0.078857421875, 1.056640625, 0.69775390625, -1.18359375, 0.7548828125, 2.744140625, -0.43310546875, 0.094482421875, -0.53857421875, -0.80029296875, 1.0185546875, -0.11175537109375, 0.83056640625, -0.386474609375, -0.58447265625, 0.2381591796875, 0.62548828125, -0.634765625, 1.2197265625, 0.403076171875, 0.25390625, 0.50390625, -0.37646484375, -0.28076171875, -0.2496337890625, 0.6474609375, 0.401123046875, 0.28125, -1.5048828125, 0.5576171875, 0.274169921875, -0.6455078125, 1.37109375, 0.7109375, 1.1767578125, 0.6025390625, 0.6240234375, 0.254150390625, 1.5009765625, -0.6875, 1.0283203125, -0.33203125, 0.191650390625, 0.6337890625, -0.032073974609375, 0.67822265625, -0.355224609375, 0.6494140625, -1.2744140625, -1.3115234375, -2.68359375, 0.4287109375, 0.347900390625, 1.5751953125, 0.72998046875, -0.71240234375, 0.72314453125, -0.708984375, 0.728515625, -1.3603515625, -0.77587890625, -0.87060546875, -0.432373046875, 0.034088134765625, -2.673828125, 1.505859375, 1.3603515625, -0.225341796875, 0.38720703125, 0.041778564453125, -0.94189453125, -0.43359375, -0.1741943359375, -0.092041015625, 0.4990234375, 0.8076171875, -0.5751953125, 0.68212890625, -0.53955078125, 0.7119140625, 1.05859375, -0.1297607421875, -1.017578125, -0.92578125, -0.095947265625, 1.3583984375, -0.371337890625, -0.10015869140625, -1.615234375, -0.487548828125, -0.1483154296875, -1.4990234375, -0.138671875, 0.295654296875, -0.58984375, -1.908203125, 0.11114501953125, -1.51953125, -0.383544921875, -0.52783203125, 0.642578125, -0.1607666015625, -2.263671875, 0.88330078125, 0.32373046875, 0.228271484375, -0.705078125, 0.5146484375, 0.0758056640625, -0.42236328125, -0.2020263671875, 0.2322998046875, 1.83203125, 0.64794921875, 0.58984375, -1.5166015625, 0.3447265625, 0.01629638671875, -1.8115234375, 1.8203125, -0.5126953125, 0.1690673828125, 0.26806640625, 0.31103515625, 0.192138671875, 1.1240234375, -0.299560546875, -1.4189453125, 0.7333984375, -0.5830078125, -0.4931640625, 0.2060546875, -1.0205078125, 0.296630859375, -0.9189453125, 0.89892578125, -0.1978759765625, -0.2359619140625, -1.3701171875, 0.76416015625, -0.71630859375, -0.435791015625, -0.283935546875, -0.4287109375, -1.1708984375, 0.60791015625, -0.0853271484375, 0.064697265625, 1.05078125, -1.2509765625, 1.201171875, -0.96826171875, -0.060516357421875, 0.6708984375, -0.517578125, 1.2001953125, -0.093994140625, 0.1387939453125, 1.0302734375, -0.65771484375, 0.2122802734375, 0.974609375, -0.3740234375, 0.6259765625, -0.423828125, 0.5732421875, -1.4677734375, 0.62646484375, -0.85693359375, 0.07720947265625, -1.396484375, -0.251953125, -0.6513671875, -0.86767578125, 0.97802734375, 1.4248046875, -1.25390625, 0.60693359375, -0.153564453125, 0.56982421875, -0.9892578125, 0.268310546875, -0.92236328125, -0.154296875, 0.034393310546875, -0.80908203125, 1.18359375, 0.65673828125, -0.0394287109375, 1.5947265625, -1.2099609375, -1.123046875, 0.07666015625, -1.6474609375, 2.16015625, 0.2325439453125, 0.57568359375, -0.285888671875, 0.5927734375, -0.97314453125, 0.1416015625, -2.337890625, -1.9326171875, 0.4326171875, 0.18798828125, -0.611328125, -0.43896484375, -3.1484375, 0.2249755859375, 0.732421875, 0.8759765625, -1.408203125, -0.306396484375, 1.208984375, -0.333251953125, 1.412109375, 1.5673828125, -0.91943359375, 1.015625, -0.98486328125, 0.46728515625, -0.6748046875, 1.1064453125, -1.42578125, -0.73046875, -1.171875, 0.27734375, -0.43212890625, -0.73388671875, 2.4140625, -1.4970703125, 0.391357421875, -0.5908203125, -1.2666015625, -1.2265625, 0.822265625, 0.58544921875, -1.1025390625, -0.258544921875, -1.4306640625, 0.400634765625, 2.271484375, -1.322265625, -0.410888671875, 1.109375, 0.41796875, 0.44384765625, -0.671875, 0.054229736328125, 1.04296875, 0.93701171875, -0.79638671875, 0.369873046875, -0.77978515625, -1.892578125, 0.17578125, -0.2208251953125, 0.88525390625, -0.94091796875, 0.1578369140625, -1.2333984375, -0.54541015625, -2.255859375, -0.7314453125, 0.2359619140625, -0.280029296875, -0.47509765625, 0.6806640625, -0.377197265625, 0.5126953125, -0.919921875, -2.484375, -1.1083984375, -0.7744140625, -0.52978515625, -0.09869384765625, -0.82861328125, -0.2076416015625, 1.583984375, -0.040313720703125, 0.41455078125, -0.130126953125, 1.931640625, -0.609375, 0.33251953125, 0.1856689453125, 0.30810546875, 0.1370849609375, 0.21533203125, 1.3037109375, 0.7802734375, 0.46142578125, 0.061187744140625, 1.283203125, 0.76708984375, 0.259521484375, -1.1484375, 1.3779296875, -0.26171875, -1.9755859375, -0.0200653076171875, 0.88134765625, -1.17578125, -1.091796875, -0.386962890625, -0.75732421875, 0.89697265625, -1.5615234375, 0.31787109375, -0.291259765625, -1.6640625, -1.5478515625, 0.49560546875, -0.35888671875, 0.432861328125, 0.841796875, -1.029296875, 0.099365234375, 0.625, -0.4384765625, 0.025482177734375, -0.93310546875, 0.72314453125, -0.056976318359375, 0.98486328125, -0.1806640625, -0.78125, 1.1513671875, -0.669921875, -1.166015625, 0.55908203125, -0.75390625, 0.342041015625, -0.68310546875, -0.994140625, 0.07257080078125, -1.4814453125, -0.370361328125, -1.5576171875, 0.2010498046875, 1.4052734375, -1.0810546875, -0.296630859375, 0.02911376953125, -1.5283203125, -1.08984375, 1.7841796875, 0.1002197265625, 0.6748046875, 0.292236328125, 0.84912109375, 1.2158203125, 0.157470703125, -0.2236328125, -0.347900390625, -1.3779296875, -0.09210205078125, -0.9150390625, 1.4892578125, -1.8203125, 0.41015625, 1.130859375, -0.158447265625, 0.09814453125, 0.262451171875, -0.1295166015625, -0.3623046875, -1.076171875, -0.296142578125, 0.467041015625, -0.98583984375, -1.1259765625, 0.6767578125, -0.69287109375, -0.70703125, -1.482421875, -0.342529296875, 0.0723876953125, -0.9169921875, -1.3173828125, 0.87646484375, 0.67724609375, 0.397216796875, 0.56396484375, 1.72265625, -0.912109375, 0.1888427734375, -0.587890625, -1.189453125, -0.52099609375, 1.115234375, -1.08984375, -3.169921875, -0.22265625, 1.400390625, -0.50146484375, 1.8798828125, 1.033203125, 0.62451171875, -0.63623046875, -1.1083984375, 0.055694580078125, -1.0244140625, 0.240234375, -0.767578125, -1.142578125, 1.8427734375, 1.302734375, 0.68212890625, -0.320068359375, -0.02587890625, -0.33203125, -0.80126953125, -0.37744140625, 1.173828125, 0.3955078125, -0.1580810546875, -1.2607421875, -0.91015625, 0.10107421875, 0.88134765625, -0.77880859375, 1.072265625, -1.564453125, -0.85791015625, -0.58740234375, 0.25146484375, 0.5302734375, -0.75927734375, -0.1978759765625, 0.2174072265625, -0.86669921875, 0.310791015625, 1.40234375, -1.4482421875, -1.529296875, -0.9931640625, -0.201904296875, -0.0123748779296875, 0.389404296875, -0.65087890625, 0.183349609375, 0.0208587646484375, -1.1416015625, 0.43994140625, -1.6240234375, 0.293212890625, -0.75048828125, -1.2607421875, -1.5078125, -0.5107421875, -0.0506591796875, 1.443359375, 0.1212158203125, 0.10064697265625, 0.71337890625, 0.44677734375, -0.93310546875, -1.2119140625, -0.107177734375, 0.044525146484375, -0.44287109375, -1.103515625, -0.3037109375, 0.75146484375, -0.5908203125, 0.9775390625, -0.06500244140625, -1.4306640625, -0.05560302734375, -0.2269287109375, 0.28125, -0.290771484375, -0.2415771484375, -0.439697265625, -0.2275390625, 0.1458740234375, 0.183349609375, -0.045013427734375, -0.76025390625, -0.485107421875, -1.1259765625, 1.248046875, 0.0239410400390625, -0.2001953125, -0.1337890625, -0.2379150390625, 0.1417236328125, 0.716796875, -2.435546875, -1.2490234375, 2.169921875, 0.15673828125, -0.21875, -0.5732421875, -0.9404296875, 0.744140625, -0.88427734375, 0.034942626953125, 1.5693359375, 1.046875, 0.85595703125, 1.41015625, -0.01155853271484375, 0.05438232421875, -0.1258544921875, -1.1806640625, 0.30322265625, 0.8740234375, 0.7685546875, -0.53857421875, 0.494873046875, 0.130859375, 0.296875, -0.438720703125, 0.09442138671875, -1.697265625, -0.438232421875, 1.0693359375, 0.11614990234375, -1.33203125, 0.12359619140625, -0.7470703125, 0.00203704833984375, -0.47216796875, 1.177734375, 1.1005859375, 0.265380859375, 1.4873046875, -0.8525390625, 1.9267578125, 0.139404296875, -0.3466796875, 1.2001953125, 0.75390625, -0.277099609375, -0.01023101806640625, 1.076171875, -0.8857421875, -0.362060546875, 0.75390625, 1.2412109375, 0.509765625, 0.5302734375, -1.08203125, 1.49609375, -0.54541015625, 1.5615234375, -1.3056640625, 0.2449951171875, -2.462890625, 0.734375, -0.96533203125, 0.03125, -0.392822265625, -0.055572509765625, -0.6650390625, -0.361572265625, 0.1007080078125, -0.457763671875, -0.79931640625, -1.439453125, -2.298828125, -2.021484375, 0.57470703125, 0.525390625, -0.1788330078125, -0.6533203125, -1.095703125, 0.332275390625, -0.2327880859375, -0.08770751953125, -0.393310546875, -0.425537109375, 0.412109375, -2.31640625, 1.1474609375, 0.52734375, 0.2237548828125, 2.71875, 0.8369140625, -0.12939453125, 0.712890625, -0.04290771484375, 0.437744140625, -0.5849609375, 0.7802734375, 1.05078125, 0.89697265625, -2.021484375, -1.658203125, 0.06341552734375, 0.5205078125, -0.93408203125, -0.00118255615234375, 0.322509765625, -1.7021484375, 0.7802734375, -0.11761474609375, 0.2236328125, -0.7275390625, -0.376220703125, 0.04498291015625, -0.49951171875, 1.1123046875, -0.052001953125, 0.031707763671875, 0.26904296875, -1.5556640625, -0.1661376953125, 0.58984375, 0.2213134765625, 1.2724609375, -1.2646484375, 0.68115234375, 0.4462890625, 0.9169921875, 2.966796875, 0.85498046875, 0.66357421875, -2.380859375, 0.86376953125, 0.94873046875, 0.120361328125, 0.022216796875, -0.054718017578125, 0.623046875, -0.0238037109375, 1.2041015625, -0.5283203125, -0.68603515625, 0.2432861328125, -0.1781005859375, 0.6787109375, 1.365234375, -1.9814453125, 0.08087158203125, 0.46240234375, -0.90185546875, -0.51123046875, 1.08203125, -0.351318359375, 1.6123046875, 1.2236328125, -0.485595703125, -1.5126953125, 0.8076171875, -0.33056640625, -0.78466796875, 0.544921875, -0.61865234375, 0.53955078125, 1.7607421875, -0.409912109375, 0.9501953125, 0.98095703125, 1.7041015625, 0.06585693359375, 0.413330078125, -0.82666015625, -0.59814453125, -0.408203125, 0.374755859375, -0.49755859375, 0.366943359375, 0.5849609375, 0.088623046875, 1.546875, 0.406005859375, -1.166015625, -1.3603515625, 0.009246826171875, 1.6728515625, 1.849609375, -1.326171875, -2.591796875, -1.0029296875, -1.3427734375, 0.405517578125, 1.8701171875, 1.4150390625, -0.55078125, -0.64794921875, -0.171875, 0.0087127685546875, 0.81298828125, 0.49853515625, 1.0419921875, -0.2294921875, 0.469970703125, -1.3427734375, -0.64013671875, 1.119140625, 0.365234375, -0.52294921875, 1.5830078125, -3.03515625, 0.375244140625, -1.267578125, -1.810546875, -0.07330322265625, 1.2255859375, -0.50146484375, -0.94970703125, 2.447265625, 1.8125, -0.72265625, 1.6083984375, -0.304443359375, 0.154541015625, -1.4658203125, 0.85791015625, 0.98193359375, -1.8544921875, 2.80859375, 1.787109375, -0.66064453125, 1.0458984375, 0.26611328125, -0.79296875, 1.419921875, -0.130126953125, -1.3447265625, -1.01171875, 1.1728515625, 0.58447265625, -1.0517578125, 0.62646484375, -1.1416015625, -0.252197265625, -0.66748046875, -1.6923828125, 0.56396484375, -0.587890625, 0.6767578125, -0.65625, 0.8671875, 0.206787109375, -1.4541015625, 0.061737060546875, -0.3076171875, 0.11614990234375, -0.270751953125, 0.016632080078125, -0.56298828125, 0.07904052734375, -0.29345703125, 0.2041015625, -1.0234375, -1.001953125, -0.055511474609375, -1.919921875, 0.108154296875, -0.10235595703125, 0.62255859375, 0.01248931884765625, 0.10943603515625, -0.85986328125, -1.2333984375, 2.087890625, 1.599609375, 0.328125, -0.0570068359375, 0.334716796875, -1.630859375, -0.65625, -0.2366943359375, -1.4541015625, 1.3466796875, 0.78369140625, -0.9013671875, 1.447265625, -0.89990234375, -0.62841796875, -1.14453125, -1.224609375, -2.30078125, 0.3046875, 0.908203125, 0.6572265625, -2.4140625, 1.958984375, -0.18359375, 0.0011854171752929688, -0.44091796875, -0.419189453125, 0.456787109375, -0.382080078125, -1.08203125, 0.84375, -1.509765625, 0.65185546875, 0.430419921875, -0.60302734375, -0.00998687744140625, 1.720703125, -0.69091796875, -0.7236328125, 0.327880859375, -0.48095703125, -0.8115234375, 0.8798828125, -0.057403564453125, -1.3291015625, 0.609375, 0.83984375, -1.8779296875, -0.88232421875, -0.25244140625, 1.689453125, 2.236328125, -0.7353515625, -0.34619140625, -1.216796875, 0.34619140625, -0.0212554931640625, 0.94482421875, 1.2392578125, 1.0078125, -0.1439208984375, 0.4365234375, -0.83740234375, -2.884765625, -0.5634765625, -0.93310546875, 0.488037109375, 0.36328125, -0.845703125, 1.181640625, -0.4443359375, -1.4228515625, -0.857421875, 0.26318359375, 1.7001953125, -0.6142578125, 0.03631591796875, 0.3330078125, 1.7080078125, 0.775390625, -0.13916015625, -0.301025390625, -0.65576171875, 0.4052734375, 0.568359375, 0.2783203125, -0.343017578125, -0.34912109375, -0.318603515625, -1.2578125, -0.2099609375, 0.187744140625, -0.63818359375, 0.425537109375, -1.119140625, -0.54638671875, -1.4365234375, -1.0791015625, 0.740234375, -0.3837890625, -0.65087890625, -0.084228515625, -1.1689453125, -0.488037109375, 0.83740234375, 1.5537109375, 1.5341796875, -1.515625, 1.029296875, 1.2451171875, 0.0244598388671875, 0.042083740234375, -1.265625, 1.767578125, 0.751953125, -1.6396484375, 2.44140625, -0.75146484375, -0.52783203125, 0.50048828125, 1.951171875, 0.30615234375, 1.6396484375, -0.09283447265625, 0.79296875, -0.405517578125, 0.634765625, -1.7080078125, 0.51220703125, -1.970703125, -0.89453125, -0.79931640625, 1.4052734375, 0.6064453125, -1.330078125, 0.66064453125, 0.56689453125, -0.69140625, -0.92041015625, -0.022674560546875, -1.12890625, 0.378173828125, 0.56396484375, -0.19775390625, 1.1240234375, -0.358642578125, -0.85302734375, 0.447509765625, 0.775390625, -0.76025390625, -0.07781982421875, -1.4326171875, 0.3154296875, 1.978515625, 2.263671875, 0.453857421875, -0.413330078125, -0.343994140625, -0.48486328125, 1.2177734375, -0.007152557373046875, -0.623046875, -1.53515625, -1.662109375, 0.1856689453125, 0.80859375, 0.24755859375, 2.4296875, -0.2783203125, -0.86572265625, 1.0673828125, -0.205322265625, 0.61962890625, 1.615234375, 1.5078125, -1.4755859375, 0.4658203125, 0.309326171875, -0.9404296875, -0.257568359375, 0.833984375, 0.04180908203125, 1.3203125, 1.19140625, -0.283203125, -2.0546875, 0.291748046875, 0.96142578125, 0.59033203125, 1.232421875, 0.662109375, 0.75244140625, -0.1256103515625, 0.50537109375, -0.091064453125, 1.5341796875, -0.8193359375, -0.64599609375, -1.8916015625, 0.329833984375, -1.3681640625, -0.2423095703125, 0.80322265625, 0.62939453125, 0.583984375, 1.150390625, -0.1280517578125, -0.478515625, -1.3544921875, -1.4599609375, 0.71142578125, -1.0830078125, 0.428466796875, 0.25439453125, -0.48046875, 0.52001953125, -0.6884765625, 0.275390625, -0.2685546875, -0.84814453125, -1.0732421875, -0.035003662109375, -0.487548828125, -0.74658203125, -1.6181640625, 1.66015625, -0.38916015625, 0.6259765625, 0.450439453125, -0.8857421875, -0.08782958984375, -0.11273193359375, -1.2197265625, -1.2841796875, 2.779296875, -0.100830078125, -0.95068359375, -0.91845703125, -1.994140625, 1.82421875, -0.708984375, 0.82470703125, 0.491455078125, -0.99609375, -0.765625, -0.498779296875, 0.1795654296875, 0.63037109375, -0.0206298828125, 1.583984375, 0.1138916015625, 0.9482421875, 0.07080078125, 1.69140625, -0.290771484375, -1.3369140625, -0.363037109375, -0.271240234375, -0.4169921875, -0.978515625, -1.0546875, -0.1435546875, 0.26611328125, 1.4599609375, -2.501953125, 0.228759765625, -0.626953125, -0.404541015625, -0.904296875, 0.3212890625, 0.1605224609375, 0.10235595703125, 1.400390625, 0.361083984375, 0.343994140625, -0.31396484375, -0.59326171875, 0.60595703125, 0.324462890625, -0.1485595703125, -0.56787109375, -0.55615234375, -0.96875, 1.2099609375, -1.8212890625, 0.0853271484375, -0.578125, -0.437744140625, 0.6875, 1.8623046875, -1.859375, -2.189453125, -1.3701171875, 0.351806640625, 0.417236328125, 0.86669921875, -0.79931640625, -0.77783203125, 1.8291015625, -1.037109375, 0.3798828125, 1.3828125, -1.37890625, 0.24072265625, -0.60693359375, 1.08203125, 0.6943359375, 0.35302734375, -0.57275390625, 0.724609375, 0.7763671875, -0.77099609375, -0.8134765625, -0.257568359375, 1.5283203125, 1.375, 1.0146484375, -0.58056640625, 0.368408203125, -0.1058349609375, 0.61962890625, -0.1090087890625, 0.9052734375, -0.2095947265625, -0.6259765625, 1.9912109375, -0.3173828125, 0.57568359375, -0.04132080078125, -0.560546875, 1.5859375, 0.0916748046875, -0.4951171875, 0.564453125, 0.1595458984375, 0.2188720703125, -0.9072265625, -1.6640625, 0.05523681640625, -0.7783203125, 0.27685546875, -1.9951171875, -0.19482421875, 0.056427001953125, -0.8642578125, -0.468994140625, 1.0244140625, -0.720703125, -0.77197265625, 1.6787109375, 0.93701171875, 0.62548828125, -1.9228515625, -0.990234375, -0.54248046875, 1.18359375, 1.1494140625, 0.16845703125, 0.8125, -0.9931640625, 0.27587890625, 0.94384765625, -0.7294921875, 1.6025390625, -0.99853515625, 0.25927734375, 0.8935546875, -0.59814453125, 0.2227783203125, -0.904296875, -0.0916748046875, 0.1536865234375, 0.95849609375, -0.429443359375, -0.8251953125, 0.578125, 0.173095703125, -0.282470703125, 0.39404296875, 0.486328125, -0.97314453125, -0.8125, 0.2349853515625, -0.00887298583984375, 0.5546875, -0.08551025390625, 0.280029296875, 0.60888671875, -0.5908203125, 0.17529296875, -0.8095703125, 0.99072265625, -0.91845703125, 0.038543701171875, 0.413330078125, 0.237060546875, -1.3447265625, -0.71533203125, -0.357177734375, 1.2021484375, 0.87255859375, 1.275390625, 1.2783203125, -0.65478515625, 1.2099609375, -0.1728515625, -0.99755859375, -2.13671875, 0.572265625, 0.313720703125, 0.324462890625, -0.1961669921875, -0.40478515625, 0.3017578125, 0.859375, 1.2744140625, 1.43359375, -1.6923828125, -0.423095703125, 0.302978515625, 0.65673828125, -0.8466796875, -0.47802734375, 0.1617431640625, 1.28515625, -0.44189453125, -0.10333251953125, -0.6767578125, 1.9482421875, -0.83642578125, -0.2037353515625, 1.896484375, -0.54052734375, 2.162109375, -0.2496337890625, 1.13671875, 1.85546875, -0.6787109375, -0.89208984375, -0.52685546875, 0.91552734375, -0.0380859375, 0.266357421875, -1.541015625, -0.791015625, -0.28759765625, -1.9755859375, -0.8916015625, -0.08062744140625, 1.0615234375, -0.499267578125, 0.0165863037109375, 1.3212890625, 2.0078125, -1.359375, 0.291748046875, 0.349853515625, -0.470703125, 1.1962890625, -0.264892578125, 0.6142578125, -0.05755615234375, -0.5390625, 0.1922607421875, -0.7412109375, -0.75537109375, 1.056640625, 0.496337890625, 0.92041015625, 0.9677734375, -1.7587890625, 1.33984375, -0.9892578125, 0.14892578125, 1.7578125, -1.001953125, -1.419921875, 0.062164306640625, -0.89306640625, -2.095703125, -1.607421875, 0.271728515625, 0.93701171875, -1.072265625, -0.165771484375, 1.037109375, 0.11553955078125, 0.2030029296875, -0.60595703125, 0.1895751953125, -1.296875, 0.323974609375, 0.62646484375, -0.10760498046875, 1.1748046875, 1.0380859375, 0.52099609375, -1.4658203125, -0.68701171875, -0.405029296875, 0.324462890625, -0.50390625, 0.0682373046875, 1.0341796875, 0.48828125, -0.8369140625, -0.1610107421875, -0.279541015625, 0.2183837890625, -0.53564453125, 0.8037109375, 0.88330078125, -1.4873046875, 0.1278076171875, -0.744140625, 0.9296875, 0.8984375, 0.5390625, 0.213623046875, -0.755859375, 0.40234375, -1.3056640625, 0.900390625, -0.27294921875, -1.21484375, 0.74609375, -0.06689453125, 1.74609375, 1.63671875, 2.671875, -0.3095703125, 0.458740234375, 1.3759765625, -0.1514892578125, 0.685546875, 0.466796875, 0.85107421875, -0.869140625, 0.56787109375, 0.306396484375, -1.3818359375, -1.3837890625, 1.2265625, 0.01538848876953125, -0.171630859375, -0.10308837890625, -0.255859375, 1.4404296875, -1.0546875, 0.3251953125, -0.478271484375, 0.9248046875, 0.363037109375, -0.86376953125, -0.0775146484375, -0.0028667449951171875, 1.13671875, -1.2587890625, -1.587890625, -1.6826171875, -0.442138671875, -1.2109375, 0.90966796875, 0.85546875, -1.228515625, 0.87548828125, -1.611328125, -0.44287109375, 0.8505859375, 0.5458984375, -0.004032135009765625, 0.421875, -0.379638671875, -0.25146484375, 3.291015625, 0.2294921875, -0.6796875, -0.05401611328125, 0.44775390625, -0.072265625, 0.433837890625, -1.1201171875, -0.8330078125, -0.5849609375, -1.6044921875, -0.499267578125, -0.943359375, -0.0272064208984375, -0.54443359375, 0.283447265625, 0.81005859375, 0.67529296875, -0.583984375, -1.0751953125, 1.2294921875, -0.047760009765625, -1.30078125, 0.61376953125, 1.1396484375, 1.1904296875, 0.2239990234375, 1.498046875, 0.06573486328125, -0.400634765625, 0.9912109375, -0.7080078125, 0.205322265625, -0.8017578125, -0.155029296875, -0.67431640625, -0.4033203125, 1.6396484375, 1.4013671875, 0.40673828125, -0.8828125, 0.385986328125, 0.00860595703125, -1.333984375, -1.376953125, 0.033721923828125, -0.3759765625, -0.66015625, 0.038604736328125, 0.73583984375, -1.2490234375, 0.70751953125, -0.20751953125, 0.41015625, 0.5126953125, 0.04498291015625, -1.16015625, 1.8701171875, -0.63134765625, -0.44921875, 1.16796875, 0.10748291015625, -0.984375, -0.44287109375, -0.796875, 1.373046875, -1.2626953125, -1.412109375, 0.5380859375, 0.1654052734375, 0.984375, -0.5927734375, 1.447265625, 0.8037109375, -0.10565185546875, 1.30078125, -0.31884765625, -0.5302734375, 0.354248046875, 1.8984375, 0.92236328125, 0.1583251953125, 1.845703125, 0.278564453125, 0.1820068359375, -0.98681640625, 0.916015625, 0.248291015625, -1.505859375, -0.427490234375, -0.46240234375, 0.5419921875, 0.4453125, 0.395751953125, 0.6953125, 0.59521484375, 0.27880859375, -1.841796875, -0.05291748046875, 0.46484375, -0.060546875, -0.9111328125, -1.001953125, -0.64404296875, -0.67333984375, 1.009765625, -1.08203125, 0.8251953125, 0.5166015625, -0.190185546875, -1.3642578125, 0.0626220703125, 2.197265625, 0.611328125, 2.701171875, -1.1279296875, -0.9248046875, -1.4853515625, 0.486572265625, -0.59228515625, 0.322998046875, 1.26953125, -0.036376953125, -0.01396942138671875, -0.291015625, 1.119140625, 0.437744140625, 1.0693359375, 1.2646484375, 0.04443359375, 0.308349609375, -1.126953125, -0.8974609375, 1.552734375, 0.6865234375, 1.443359375, -0.744140625, 0.61865234375, -0.611328125, -1.37890625, 1.072265625, 0.4345703125, 0.394775390625, -0.841796875, 0.42724609375, 1.1494140625, -2.103515625, -1.0341796875, -0.31787109375, -0.70556640625, -0.220458984375, 1.560546875, -1.2255859375, 2.21875, -1.419921875, 0.3125, 1.1044921875, 0.8740234375, -0.045867919921875, -0.80419921875, 0.2352294921875, 0.093994140625, -1.2822265625, -1.6181640625, 0.31689453125, -0.2138671875, -1.33203125, 2.125, 1.5322265625, 0.8271484375, 0.345947265625, -0.57080078125, -0.362060546875, -1.17578125, 0.6845703125, -0.158447265625, -1.1025390625, -0.64208984375, -0.301513671875, -1.4443359375, -1.228515625, 1.2158203125, -3.318359375, 0.63525390625, 1.0751953125, -1.8798828125, 0.74072265625, -1.7919921875, -2.67578125, -0.0166168212890625, 0.99365234375, -0.2283935546875, -0.51171875, 0.56591796875, -0.045684814453125, -0.0136260986328125, -1.802734375, 0.10833740234375, -0.08758544921875, -0.18212890625, 1.6708984375, -0.490478515625, -0.447509765625, -1.2744140625, 0.1826171875, 0.09808349609375, 0.30126953125, 0.78857421875, -1.8623046875, -1.8095703125, -1.4580078125, -0.78125, 0.73974609375, 0.276611328125, -0.40283203125, 0.2919921875, -1.326171875, 0.41552734375, 0.642578125, 0.0087127685546875, -0.415771484375, -0.97021484375, 0.60498046875, -0.52685546875, 0.8662109375, -0.161376953125, -0.1412353515625, 1.888671875, 1.736328125, 0.291259765625, -1.5615234375, 0.0027561187744140625, -0.373046875, 0.7412109375, -0.1715087890625, -0.226318359375, -0.36865234375, 1.03125, 0.329833984375, -0.1898193359375, -1.41796875, -0.6396484375, 1.03515625, 0.33203125, -0.79296875, -1.7138671875, -0.18212890625, -0.54541015625, 0.79296875, -1.8310546875, -0.80859375, 0.83251953125, 0.383544921875, 1.02734375, 1.0224609375, -1.1181640625, -0.0309906005859375, 0.08331298828125, -0.93603515625, 1.80859375, 1.4296875, 0.591796875, -0.76611328125, 0.148681640625, -0.76318359375, 0.396484375, 0.49365234375, 0.12060546875, 1.0244140625, -1.4560546875, -0.509765625, 1.201171875, -0.6484375, 0.4453125, -0.133544921875, -0.2005615234375, -0.66162109375, -1.705078125, 1.078125, -0.94775390625, -0.2822265625, 0.0677490234375, 0.2059326171875, 1.0263671875, -0.7880859375, -0.7109375, 0.8408203125, -0.4345703125, -0.77783203125, 0.70556640625, -0.5302734375, -0.1961669921875, 0.908203125, -0.322509765625, -1.6728515625, 0.441650390625, -0.2353515625, 1.5576171875, 0.186767578125, 0.6015625, -0.47705078125, -0.7412109375, 1.9990234375, -0.14599609375, -1.431640625, -0.285400390625, 1.349609375, 0.7509765625, -0.31494140625, 0.9169921875, -0.93701171875, -1.005859375, 1.298828125, 0.5791015625, -0.728515625, 0.1624755859375, 0.85205078125, 0.484619140625, -0.99951171875, 1.3603515625, -2.7109375, -0.3701171875, 1.787109375, -0.2412109375, -0.452880859375, 1.2958984375, 0.64599609375, -0.215087890625, -0.84814453125, -1.6884765625, 2.830078125, 2.0859375, -0.3310546875, 0.86328125, -1.7197265625, 0.280517578125, 0.1318359375, -0.66162109375, 0.37646484375, -1.328125, -0.697265625, -0.26123046875, -0.274658203125, -1.083984375, 1.025390625, -0.84716796875, 1.2841796875, -0.7548828125, -0.6328125, -0.00893402099609375, -0.91650390625, 0.9453125, -2.162109375, -1.2841796875, -0.480224609375, -2.5625, 0.122314453125, -0.80078125, 2.3671875, -0.05694580078125, -0.30712890625, -0.43310546875, -0.560546875, -1.1728515625, 0.74072265625, -0.476318359375, 0.81591796875, -0.458984375, -0.77001953125, -0.435302734375, -0.9697265625, 0.5361328125, 0.2177734375, -0.240478515625, -1.7080078125, -0.5654296875, 0.2880859375, -0.31005859375, -0.921875, -0.429931640625, -1.5263671875, 0.38427734375, 1.140625, 0.298583984375, 0.2008056640625, -0.35791015625, 0.0054779052734375, -0.10296630859375, 0.80078125, -0.9052734375, -0.2171630859375, 0.3115234375, 0.78271484375, -2.46484375, 0.67626953125, 0.69921875, 1.2802734375, 0.10986328125, 1.0576171875, -0.5263671875, -0.7998046875, 0.48828125, -1.0693359375, -0.317626953125, 1.603515625, 0.60107421875, -0.442626953125, 0.87744140625, -0.0933837890625, 1.7890625, 0.8203125, -0.54345703125, -0.6005859375, -0.54150390625, 1.208984375, -0.39599609375, 1.974609375, -1.4931640625, -1.236328125, 0.321533203125, -0.51806640625, -0.1640625, -1.8984375, 0.8974609375, -1.25390625, -0.3232421875, -0.37841796875, -0.07098388671875, -1.630859375, 1.240234375, 1.84375, 1.8173828125, 0.78125, 1.4345703125, 0.578125, -0.6328125, -0.298095703125, -0.429931640625, -0.20751953125, -0.343505859375, 0.016815185546875, 0.373291015625, -0.71630859375, -1.755859375, 1.66015625, 0.87353515625, 0.85888671875, 0.7314453125, 1.169921875, 1.029296875, 2.650390625, -0.416748046875, -0.01041412353515625, -0.078369140625, -0.55029296875, -0.144775390625, 0.0570068359375, -1.6298828125, 1.7392578125, -0.407470703125, -0.99462890625, -0.49853515625, 0.168701171875, -1.119140625, 0.180908203125, -0.197509765625, -0.63427734375, 1.2255859375, 2.34765625, -0.3525390625, -1.029296875, -1.2314453125, -0.61865234375, 2.044921875, 0.422119140625, -1.64453125, 0.51611328125, 0.9169921875, -0.90478515625, 0.185302734375, -0.7900390625, 0.425048828125, -0.900390625, -0.794921875, 0.246826171875, -0.78271484375, 1.1298828125, -0.724609375, -0.2042236328125, 0.2425537109375, -0.89453125, -0.456298828125, 1.2314453125, -0.2432861328125, -0.1868896484375, 0.46484375, 1.25390625, 0.157958984375, -0.8583984375, 1.2724609375, -0.859375, -1.484375, 0.1666259765625, -2.322265625, -1.794921875, -0.70654296875, -0.1705322265625, -0.08740234375, -0.1358642578125, -1.3359375, -0.8857421875, 0.7763671875, 1.609375, 0.317138671875, 0.412353515625, 0.0302734375, 0.68798828125, -0.873046875, 0.225341796875, -0.25341796875, 0.94677734375, 0.092529296875, 0.42578125, -0.2161865234375, -0.0237579345703125, 1.6962890625, 0.55517578125, 0.1512451171875, -0.467041015625, 0.48583984375, 0.472900390625, 1.0361328125, 0.0287017822265625, 0.7763671875, -1.8798828125, 0.478759765625, -1.6015625, -0.34326171875, -0.96337890625, 1.244140625, 0.666015625, -1.9423828125, 0.94482421875, -0.5185546875, -0.96923828125, -0.333740234375, 0.06536865234375, -0.6484375, -1.8359375, 1.296875, -0.2169189453125, -1.9931640625, -1.1787109375, -0.94189453125, -0.625, -2.123046875, 0.76220703125, -0.89013671875, 1.1826171875, 0.56689453125, -1.69140625, -0.370361328125, -1.1201171875, -0.03564453125, -0.74267578125, 1.697265625, 0.06365966796875, -0.8642578125, 0.033843994140625, 0.1697998046875, -0.149658203125, 0.603515625, 0.391845703125, 1.0146484375, -2.1484375, 0.5673828125, 1.484375, 0.139892578125, 0.117431640625, 0.84326171875, -0.34033203125, 0.474365234375, 0.2197265625, -2.201171875, -0.88916015625, 0.471923828125, -0.422607421875, 0.73876953125, -1.0361328125, -1.0986328125, -1.498046875, -1.916015625, -0.9619140625, -0.1739501953125, 0.150146484375, 0.468505859375, 0.1595458984375, 0.334228515625, 0.45361328125, 0.86328125, 0.261474609375, -1.7890625, -1.6806640625, 0.791015625, -0.5859375, 0.076904296875, -1.669921875, 0.796875, 0.042510986328125, -0.2186279296875, 0.205322265625, -0.16796875, -1.3994140625, -0.390380859375, 1.140625, -0.60009765625, -0.9794921875, -0.384033203125, 0.505859375, 2.197265625, -1.0849609375, 1.55859375, 1.087890625, -0.99169921875, -0.0171661376953125, -0.849609375, -1.4326171875, -0.1351318359375, -1.3818359375, 1.18359375, 0.72900390625, -0.9853515625, 1.9287109375, 0.87646484375, -1.05078125, 0.33203125, 0.53857421875, 1.8837890625, -0.73486328125, 0.36328125, -0.333984375, 0.61767578125, 1.234375, 0.84765625, -0.84423828125, -0.62646484375, -1.8916015625, -0.5556640625, 0.87548828125, 0.3994140625, 0.654296875, 0.053314208984375, 0.95458984375, 0.59375, 0.583984375, 1.3271484375, -0.345947265625, 0.8974609375, 0.0953369140625, 0.56787109375, -0.89892578125, -0.51806640625, -1.271484375, -0.904296875, 1.0927734375, 0.035736083984375, -0.7470703125, -0.1058349609375, 0.293212890625, -1.65625, 0.61181640625, 0.2086181640625, 0.7783203125, -0.0328369140625, -0.183837890625, 0.77099609375, 0.2386474609375, 0.6572265625, 0.97509765625, 1.419921875, 0.53955078125, -0.994140625, -0.09844970703125, 0.7763671875, 1.6162109375, -0.113037109375, -0.50537109375, 0.8974609375, -1.107421875, 0.64501953125, 0.478515625, -0.46435546875, -1.4853515625, 0.9287109375, 0.6103515625, 1.5126953125, 0.2459716796875, 1.048828125, 2.072265625, 0.08697509765625, 0.07574462890625, 1.2705078125, -0.276123046875, 0.07879638671875, 1.7607421875, 0.861328125, -0.45849609375, 0.359375, -1.4443359375, 1.453125, 1.416015625, -0.73583984375, 0.1917724609375, -0.289794921875, -1.083984375, -0.406982421875, 1.0498046875, -0.6455078125, -0.6259765625, -1.0712890625, -1.9306640625, -0.78125, 0.08160400390625, -0.2802734375, -0.8330078125, -1.5478515625, -0.116943359375, -0.32177734375, 1.00390625, -0.40185546875, 0.98779296875, -1.8359375, 1.25, 0.87744140625, 0.7880859375, -1.4248046875, 0.004169464111328125, -0.062744140625, 0.056976318359375, -0.091064453125, 2.328125, -1.3828125, -0.85595703125, -0.79736328125, 1.4921875, 0.117431640625, 2.638671875, -0.0036067962646484375, 0.06622314453125, -0.53466796875, 1.4755859375, 1.8486328125, 0.474365234375, 0.654296875, 0.55224609375, -0.1507568359375, -1.072265625, 0.411865234375, -0.383056640625, -0.342041015625, 0.08154296875, 0.52099609375, 0.249755859375, 0.59619140625, 0.7060546875, -1.0458984375, -0.4306640625, -0.44091796875, -1.1044921875, -1.5751953125, -0.90283203125, -1.38671875, -0.95849609375, -0.55322265625, 0.1751708984375, -0.031951904296875, 0.87109375, -1.2265625, -1.8544921875, -0.0181884765625, -1.0751953125, -0.09234619140625, 0.79443359375, -0.564453125, 0.12408447265625, -2.4140625, -0.71484375, 0.97900390625, 0.01058197021484375, 2.099609375, 0.059783935546875, 0.72412109375, -1.5849609375, 0.29296875, 1.2099609375, -0.93408203125, -0.00891876220703125, 1.103515625, 0.2734375, -0.5556640625, -0.4736328125, 0.266357421875, 0.041351318359375, -0.228759765625, -0.65283203125, -0.8486328125, -0.11126708984375, 1.314453125, -2.4765625, -0.85791015625, 0.4873046875, 0.299560546875, 0.5361328125, -0.480712890625, 0.53515625, 1.0810546875, -2.4140625, 0.1055908203125, 1.845703125, 2.28515625, -0.21630859375, 0.342041015625, 1.5126953125, 0.74658203125, -0.312255859375, 0.6787109375, -0.044036865234375, 1.2626953125, -0.66552734375, -0.26708984375, -0.72314453125, 2.9921875, 0.470947265625, 1.435546875, 0.0689697265625, 0.0194091796875, 2.0234375, 0.4267578125, -0.476318359375, 0.434326171875, -0.62255859375, 0.873046875, 0.1392822265625, -0.10443115234375, -1.482421875, -0.619140625, 0.49560546875, -0.86376953125, -0.14794921875, -0.6474609375, -0.422119140625, 1.7255859375, 0.65087890625, -1.0625, -0.94873046875, 0.45703125, -0.161865234375, -0.63037109375, -1.3388671875, 0.89111328125, 0.61962890625, 1.5625, 0.6298828125, 0.76220703125, 2.11328125, 0.32177734375, -0.486083984375, 0.130615234375, -1.2421875, -1.013671875, 1.443359375, -0.30810546875, -0.6689453125, -0.344970703125, 1.94921875, 0.8173828125, 1.1904296875, -0.008575439453125, 0.04296875, 1.1748046875, -0.23486328125, -0.75048828125, 0.7490234375, -0.340087890625, 1.0791015625, 1.265625, -1.0458984375, 1.2080078125, -1.0625, -1.087890625, -0.95849609375, 1.4013671875, 0.49951171875, -1.798828125, -1.466796875, 1.1728515625, -0.689453125, -0.159912109375, -1.001953125, -1.0654296875, -0.0731201171875, 0.36962890625, 1.037109375, -0.277099609375, 0.328125, 0.96533203125, -0.126220703125, -0.0367431640625, -0.71923828125, 1.2333984375, 0.080810546875, -0.79638671875, -0.03350830078125, 0.1248779296875, -0.448486328125, 0.040435791015625, 0.1920166015625, 0.1221923828125, 0.5869140625, 0.62646484375, -0.2403564453125, -1.3935546875, 1.1416015625, -0.1292724609375, 0.85546875, 0.69091796875, 0.90576171875, 1.734375, -0.67041015625, -0.537109375, -0.376953125, -0.048126220703125, 0.421142578125, 1.23046875, 2.224609375, 1.5185546875, 1.1357421875, -1.587890625, -0.91259765625, 1.087890625, -0.19287109375, -2.216796875, -0.300537109375, 1.2373046875, 1.3486328125, 0.6181640625, -0.1995849609375, -1.0546875, -0.83984375, -0.081298828125, -1.0068359375, -1.994140625, 1.1533203125, -0.529296875, -1.49609375, -0.13525390625, -0.64990234375, 0.78125, 0.91357421875, 0.3564453125, 0.0799560546875, 0.2734375, 0.116455078125, 1.6240234375, -1.021484375, 1.2451171875, -2.30078125, 1.1669921875, 0.59912109375, 0.08551025390625, 0.2003173828125, -0.7216796875, 1.12109375, 1.193359375, -0.732421875, -2.181640625, 0.2783203125, 0.279052734375, 1.697265625, -1.09375, -1.0810546875, 0.95458984375, -0.426513671875, 1.71484375, 2.01953125, 0.1795654296875, 2.171875, -0.80029296875, 0.53271484375, -0.43115234375, -2.28515625, -0.408935546875, 0.309814453125, 1.041015625, 0.95068359375, -2.95703125, -0.2213134765625, 1.2294921875, -0.919921875, 0.0154876708984375, -2.0234375, 0.755859375, 0.00722503662109375, -0.1673583984375, 1.08203125, -1.287109375, -0.55322265625, 0.334228515625, -0.54443359375, 0.87841796875, -1.3671875, -0.260986328125, 0.376953125, 1.4306640625, -0.67138671875, 2.248046875, 0.6123046875, 0.705078125, 0.47265625, -0.63818359375, 0.5732421875, -0.3955078125, 0.397705078125, 0.12457275390625, 0.5947265625, -0.73779296875, -0.587890625, -0.67333984375, 0.97509765625, 1.0185546875, -0.313232421875, 0.443603515625, 0.1793212890625, 0.045196533203125, -1.291015625, 0.0780029296875, -0.336669921875, 0.046356201171875, 0.313720703125, -0.474365234375, 1.15234375, 0.470947265625, 0.46435546875, -0.431640625, 0.302734375, 1.669921875, 3.0390625, -0.1396484375, -0.055938720703125, -0.68359375, 0.296875, 0.1571044921875, -0.3369140625, 0.677734375, -0.71826171875, -0.75390625, -1.2275390625, -0.2939453125, 0.93115234375, -0.822265625, -0.681640625, 0.375732421875, 1.37890625, -0.263671875, 0.74755859375, 0.61572265625, -0.5341796875, 0.60546875, -0.169677734375, 0.08544921875, -1.107421875, -0.77392578125, 1.1357421875, 0.34619140625, 1.359375, -0.10821533203125, 0.99853515625, -2.576171875, 0.419677734375, -1.654296875, 1.541015625, 0.462158203125, -0.196533203125, 1.806640625, 1.3154296875, 1.265625, -0.317138671875, -0.283203125, 0.7587890625, 0.335205078125, 1.0810546875, 0.80126953125, 0.10345458984375, 0.30615234375, 0.367919921875, -0.93408203125, 1.1123046875, -1.56640625, -1.142578125, 0.0791015625, 0.80517578125, 0.480224609375, -0.377685546875, 1.5107421875, -0.214599609375, 1.662109375, 0.91064453125, 0.7158203125, -1.587890625, 0.5302734375, 0.4580078125, 0.6435546875, 1.36328125, -0.1002197265625, 0.397705078125, -0.49658203125, 0.72021484375, 1.140625, -0.27197265625, 0.351806640625, 0.326171875, -1.1142578125, 0.20654296875, 1.3203125, -1.43359375, 0.289306640625, 0.49169921875, -1.591796875, -1.7529296875, -2.5234375, 0.243408203125, 0.11065673828125, -0.32080078125, 1.2158203125, -0.07879638671875, -1.1513671875, 0.43798828125, 0.09368896484375, 1.02734375, -2.984375, 1.59375, 0.6650390625, -0.410400390625, 0.324951171875, -0.046661376953125, 1.6396484375, -0.09625244140625, -0.5849609375, -0.1505126953125, -0.76123046875, 1.5654296875, -1.146484375, 0.251220703125, 1.5234375, -0.828125, -0.058868408203125, -0.166015625, -1.3349609375, 1.0068359375, 0.5634765625, 1.0078125, -1.890625, 2.41796875, -1.3125, 1.7265625, 1.2548828125, 1.130859375, -0.8798828125, 2.41796875, -0.457763671875, 1.2587890625, 0.55908203125, 0.200439453125, -0.463134765625, -0.41015625, -0.343505859375, -0.63427734375, -0.4814453125, 1.0478515625, -0.3896484375, -0.017303466796875, -0.76123046875, -1.10546875, -0.58203125, 1.4765625, 0.962890625, -0.70947265625, 2.96875, -0.99560546875, -1.0546875, -0.07647705078125, -1.8193359375, 0.82861328125, 0.9111328125, 0.59375, 0.2822265625, 0.1859130859375, -0.5068359375, -1.6337890625, -1.1591796875, -1.6171875, -2.3515625, 1.0751953125, 1.2861328125, 0.16259765625, 1.0263671875, -1.470703125, -2.576171875, -0.99609375, 0.48681640625, 1.341796875, -0.1563720703125, 0.5390625, 0.9658203125, 1.1650390625, -0.122314453125, 0.49951171875, -0.68701171875, -0.1339111328125, -0.92333984375, -0.88818359375, -0.564453125, 2.142578125, 0.61328125, 0.7001953125, -2.322265625, -1.05078125, -1.1669921875, 0.444580078125, -0.0660400390625, 0.14990234375, -0.43505859375, -1.806640625, -1.73828125, -0.12335205078125, -0.81689453125, -0.37890625, 0.5498046875, 0.92041015625, 0.0007147789001464844, -0.4208984375, -1.9609375, 0.1224365234375, 1.9208984375, -1.29296875, -0.01033782958984375, -1.9931640625, 0.176025390625, -0.40087890625, 1.6806640625, -0.0838623046875, 0.08270263671875, -1.2119140625, 0.568359375, -1.0380859375, 1.2294921875, -0.71337890625, -1.115234375, -1.330078125, -1.5830078125, -2.1328125, 0.765625, 0.0154266357421875, 1.07421875, 0.64697265625, 1.044921875, 0.141357421875, -1.275390625, 0.372314453125, 1.259765625, -0.3359375, 0.53173828125, 0.50390625, 0.0899658203125, -0.578125, -0.3837890625, 0.481201171875, 0.56396484375, -0.6396484375, 0.51318359375, -0.73974609375, -1.6640625, 0.80859375, 1.1484375, 0.129150390625, 0.77685546875, 1.6875, -0.93505859375, -0.65087890625, 0.97607421875, -0.277587890625, 1.46484375, 0.17333984375, -0.6044921875, -0.8408203125, -2.2890625, 0.53564453125, -0.42626953125, -1.5966796875, -0.6328125, 0.36474609375, 1.1552734375, -1.9150390625, -0.0290069580078125, 0.57177734375, -1.14453125, 1.0234375, -0.2135009765625, -0.71142578125, -3.41796875, -0.1405029296875, 2.03515625, -0.6328125, -0.050262451171875, 1.640625, -0.2213134765625, -0.257080078125, 2.734375, 0.0777587890625, -0.1768798828125, -0.8779296875, 1.09375, 1.2685546875, 1.2412109375, -0.3974609375, -0.193603515625, 0.9208984375, -1.0546875, -0.472412109375, -1.490234375, 0.1207275390625, -1.1005859375, -0.26953125, -0.767578125, 0.79736328125, 0.810546875, 1.2333984375, -0.364501953125, -0.457275390625, -0.0765380859375, -0.58056640625, -0.1292724609375, -1.111328125, -0.50390625, -0.014251708984375, -0.0791015625, 2.04296875, 0.70947265625, -0.705078125, -1.2724609375, 0.34423828125, -0.8974609375, 0.74951171875, 0.4599609375, 0.8837890625, 0.65185546875, 1.2763671875, 0.239990234375, -0.2166748046875, -1.24609375, -0.91162109375, -0.377197265625, -0.61279296875, -0.8916015625, -0.323974609375, 0.76611328125, -1.3720703125, -0.61328125, 0.7294921875, 0.51513671875, 1.1787109375, -1.935546875, 1.8984375, -0.35107421875, 0.487060546875, 0.8642578125, 1.9453125, -0.576171875, 0.24755859375, -1.0, -2.54296875, 0.68310546875, -2.296875, -1.5654296875, -0.6572265625, 1.4560546875, -1.0048828125, -0.4248046875, 0.8681640625, -0.8408203125, -0.96484375, -0.418701171875, 0.70556640625, -0.491455078125, 1.0537109375, 0.58447265625, 0.422607421875, 0.9501953125, 0.2391357421875, -1.298828125, 1.21875, 0.7509765625, 0.4794921875, 0.311279296875, 1.8125, -0.446044921875, -0.705078125, -0.2333984375, 0.88818359375, -1.1357421875, -0.525390625, -0.3505859375, 0.75146484375, 0.4052734375, -0.50341796875, -0.892578125, 1.546875, -1.802734375, -0.2403564453125, -0.73779296875, 0.372802734375, 0.66650390625, 0.363525390625, 0.181640625, -0.266357421875, 0.09417724609375, 0.74951171875, -0.78125, 0.41064453125, -1.216796875, -1.005859375, -0.225341796875, 0.50927734375, -0.242431640625, 0.7138671875, -0.390869140625, 0.8388671875, -0.29052734375, 0.5908203125, -0.11846923828125, 0.525390625, 0.81884765625, 0.495361328125, 0.80615234375, 1.3779296875, -1.1337890625, -1.01171875, 1.3076171875, -0.449951171875, -0.91259765625, 0.90966796875, 1.763671875, 0.08978271484375, 0.7255859375, 0.48046875, 0.0156402587890625, -1.2890625, 1.1328125, 1.3291015625, -0.63427734375, -0.312744140625, -0.8251953125, 1.3046875, -0.5634765625, 0.0989990234375, -0.480224609375, 1.0869140625, 0.87744140625, -0.76611328125, 0.0657958984375, -0.876953125, 0.9404296875, 0.009796142578125, -0.59765625, -0.270751953125, 0.775390625, 0.198486328125, 0.300537109375, 1.478515625, -0.1959228515625, 0.64404296875, 0.52001953125, 0.51513671875, 0.53515625, 0.351806640625, 1.1669921875, 0.408447265625, 0.830078125, -1.2265625, -1.7890625, -0.08123779296875, -1.181640625, -0.01470184326171875, -0.5322265625, -0.06396484375, -1.849609375, 0.75634765625, -0.44091796875, 1.283203125, 1.46484375, -1.060546875, -1.2763671875, 3.087890625, 0.8564453125, -0.44091796875, -0.82421875, -0.1407470703125, 1.7587890625, 0.388427734375, -2.642578125, 1.310546875, -0.86474609375, -0.36865234375, 1.580078125, 0.13916015625, -0.0105133056640625, 0.130859375, -0.74267578125, -0.480224609375, 0.06829833984375, 1.4931640625, -0.4921875, 0.1890869140625, 0.78466796875, 0.1053466796875, -0.73779296875, -0.72314453125, 0.26953125, -1.220703125, 0.180419921875, -0.47802734375, 0.04486083984375, -0.344970703125, -0.1807861328125, -0.7880859375, -1.7109375, 0.76904296875, -0.09619140625, -1.2744140625, 0.320068359375, -0.387451171875, -1.3759765625, -0.69140625, -0.9677734375, -0.61279296875, -1.181640625, 0.1541748046875, 0.71533203125, 0.3642578125, -0.42724609375, 0.0875244140625, -1.3212890625, -0.953125, 0.5263671875, 1.9951171875, 0.199951171875, -0.1383056640625, 2.154296875, 1.7822265625, 1.228515625, -0.05078125, -0.408203125, 0.50537109375, 0.46826171875, -0.192138671875, -1.990234375, 0.05084228515625, 0.40380859375, 2.15625, 1.919921875, -0.767578125, 0.99365234375, 0.11724853515625, 0.033233642578125, -0.1357421875, 1.0419921875, -1.2109375, 0.344482421875, -1.26953125, -0.0582275390625, -0.6640625, -3.08203125, -0.55419921875, 0.98291015625, 0.9013671875, 1.3271484375, -0.1102294921875, -0.89453125, 0.0154876708984375, 0.72900390625, -0.83837890625, 0.69384765625, -0.50146484375, 0.568359375, -1.2548828125, 1.5234375, -0.521484375, 1.0166015625, 0.254150390625, -0.2396240234375, 0.68017578125, -0.74560546875, 0.0977783203125, -1.7607421875, -0.5107421875, 1.216796875, -0.039642333984375, -0.2093505859375, -0.2264404296875, -0.97021484375, 0.152099609375, 0.03240966796875, 1.5, 1.611328125, -0.82080078125, -0.33544921875, -1.0029296875, 0.14990234375, -0.7958984375, 0.7099609375, -0.1309814453125, 0.5888671875, -0.173583984375, 0.043914794921875, -0.56005859375, 0.986328125, -1.7001953125, -0.1346435546875, 1.0048828125, -1.052734375, -0.8154296875, -1.0283203125, 0.09228515625, -0.6962890625, 0.125732421875, -0.94873046875, -2.27734375, 0.5302734375, 0.73046875, -0.492431640625, 0.34375, -2.18359375, 0.09649658203125, -0.28076171875, 1.9072265625, -1.9599609375, -0.447509765625, -0.418212890625, -0.337890625, -0.12109375, -0.26123046875, -0.213134765625, 0.64599609375, 0.440673828125, 1.4443359375, -0.169677734375, 1.1533203125, 0.054046630859375, -0.5947265625, -1.00390625, 0.2734375, -0.5380859375, -0.396728515625, -0.7783203125, -0.307373046875, 1.267578125, -0.290771484375, -0.7109375, 0.56689453125, -0.7119140625, 0.146484375, 0.130859375, 1.58984375, 1.4990234375, -0.78076171875, -0.05096435546875, -0.958984375, -0.4873046875, 0.548828125, 0.36328125, -0.76025390625, 1.7646484375, -0.89306640625, 0.7236328125, -0.84375, 0.65869140625, -0.1676025390625, 1.537109375, 0.71484375, 2.2890625, -0.339599609375, 1.361328125, 0.0841064453125, -1.15234375, -0.798828125, 1.2119140625, 0.9111328125, -0.341064453125, -0.53564453125, 1.421875, 0.810546875, -0.1695556640625, -1.216796875, -0.912109375, -1.9931640625, -2.39453125, -0.7080078125, -1.6982421875, 0.9052734375, -1.2470703125, 0.320068359375, -1.04296875, -0.61328125, -0.1668701171875, 0.41943359375, 0.17529296875, 1.224609375, -0.2890625, -0.41064453125, -0.1072998046875, -1.3212890625, 0.06939697265625, 1.10546875, 0.98095703125, 0.98974609375, 1.0908203125, 0.496826171875, -0.1524658203125, 1.3388671875, 0.666015625, 1.2109375, -1.373046875, -0.08050537109375, -0.59814453125, 0.71484375, -0.130126953125, 0.34130859375, 0.1368408203125, -1.3251953125, 1.095703125, -0.41259765625, -0.6689453125, -0.027984619140625, 0.400634765625, -0.05767822265625, -0.1822509765625, -0.55078125, -0.59912109375, -0.62939453125, -1.50390625, -0.82470703125, 0.0726318359375, -0.65380859375, 0.1983642578125, 1.865234375, 0.88134765625, -0.7568359375, -0.12384033203125, -0.17431640625, -0.294677734375, 1.826171875, -0.07293701171875, -0.52734375, -0.55322265625, -0.25, -1.6005859375, 1.095703125, -0.08477783203125, -0.8017578125, 1.34375, -0.92529296875, 0.455322265625, -2.15234375, 0.3349609375, -1.0068359375, 0.38330078125, 1.841796875, 1.17578125, 2.76171875, -1.564453125, 0.7275390625, 0.237060546875, -0.024871826171875, 1.646484375, 0.057769775390625, -0.79638671875, -1.181640625, -0.32470703125, 0.4443359375, -0.96484375, -2.970703125, -0.9609375, -0.82275390625, -0.2462158203125, 0.1722412109375, 0.55615234375, -0.162841796875, -2.880859375, 0.9912109375, -2.306640625, -0.315185546875, -0.1673583984375, 1.9892578125, -0.720703125, 0.08648681640625, -1.802734375, 1.4052734375, 2.169921875, -1.8095703125, 1.990234375, 0.343505859375, -0.87548828125, -1.57421875, -0.10772705078125, -0.4775390625, -2.462890625, 1.0322265625, 0.202880859375, -1.2724609375, 0.1494140625, -0.2176513671875, -1.03125, -0.471923828125, -0.5205078125, 1.078125, -0.3447265625, -1.5595703125, 0.0703125, 0.2127685546875, -0.29150390625, -0.1729736328125, 0.0263519287109375, 1.46875, -0.110595703125, 0.23095703125, 1.025390625, -0.34521484375, -1.5419921875, 0.5146484375, -1.2919921875, 1.501953125, 0.94482421875, -0.50830078125, -0.2149658203125, -0.35888671875, -1.4794921875, 0.4267578125, 0.0037021636962890625, -0.463623046875, 1.720703125, 1.10546875, -0.39111328125, 1.7080078125, -0.046875, 0.2484130859375, 1.365234375, -0.8505859375, -0.424072265625, -0.054168701171875, -0.4990234375, 1.5009765625, 0.86572265625, -0.40087890625, -0.55224609375, 1.6826171875, 0.52734375, -1.6884765625, -1.3876953125, -0.31494140625, -0.4462890625, -2.34765625, 0.7783203125, -0.019500732421875, -0.568359375, -0.564453125, -0.278076171875, 1.142578125, -0.697265625, 1.0107421875, 0.91748046875, 1.345703125, -1.31640625, 0.6181640625, 0.35546875, 0.79052734375, -1.4365234375, 0.17333984375, -0.351806640625, -1.314453125, -1.16796875, 0.312744140625, 2.224609375, -0.26708984375, 0.2607421875, 0.673828125, -0.61328125, -0.02337646484375, -0.347412109375, 0.256103515625, -1.2431640625, 0.9931640625, 0.058074951171875, -1.380859375, 0.411376953125, 0.525390625, 0.10546875, 1.818359375, 1.0869140625, -0.296630859375, -0.5634765625, 0.05731201171875, -1.3193359375, -1.2353515625, -0.483154296875, 0.58935546875, 0.56884765625, -1.0322265625, 1.1103515625, -1.5224609375, -0.55908203125, 0.164794921875, -0.92431640625, -0.7353515625, 0.82666015625, 0.72607421875, 1.0859375, 0.0207672119140625, -0.1611328125, -0.183349609375, 0.94482421875, 0.6494140625, -0.01654052734375, -0.234375, -0.445068359375, -0.404541015625, -0.7509765625, -1.345703125, 0.7529296875, 2.60546875, -0.283935546875, 1.6669921875, -1.9453125, 0.1539306640625, -0.64599609375, -0.436767578125, -0.65576171875, -0.326171875, 1.8291015625, 0.60498046875, -0.56982421875, 1.212890625, 0.90771484375, 1.2275390625, 0.325439453125, -1.375, -1.5517578125, -1.171875, -0.7529296875, 0.17822265625, 0.396240234375, 1.0849609375, -0.85888671875, -0.006000518798828125, -2.205078125, 1.703125, 1.53515625, -0.92431640625, -0.72509765625, 1.208984375, 0.8740234375, -0.0882568359375, 0.6484375, 0.05029296875, -1.3134765625, -0.5830078125, 1.2900390625, 1.2216796875, 0.18701171875, 1.31640625, 0.72314453125, -0.55859375, 0.37548828125, 0.023162841796875, -0.2271728515625, -1.9951171875, 0.96142578125, 0.0216522216796875, 1.5673828125, -1.6044921875, 1.2724609375, -0.483154296875, 2.18359375, -0.3369140625, 1.0576171875, -0.75341796875, -0.0219879150390625, 0.416015625, 0.03564453125, 0.053497314453125, 0.51123046875, -3.015625, 0.6435546875, -0.448486328125, 0.387451171875, -1.466796875, -0.403564453125, 2.11328125, -1.0634765625, 0.78759765625, 0.66796875, 1.11328125, -1.5810546875, -0.89111328125, 0.21044921875, 1.21484375, 0.771484375, -1.724609375, -0.51025390625, 0.06793212890625, -0.767578125, -0.8427734375, 0.294677734375, -0.52587890625, -1.6572265625, 0.74462890625, 0.52880859375, 0.483642578125, -1.1044921875, -0.42529296875, 1.5615234375, 0.88232421875, -0.76123046875, 0.49072265625, -1.8623046875, 0.50537109375, 0.6376953125, -0.259521484375, 0.2880859375, 1.1875, -0.65380859375, 1.4638671875, 1.603515625, 1.0205078125, -0.95263671875, 0.461181640625, 1.0654296875, -0.6533203125, -0.7802734375, -0.5478515625, 0.97607421875, 0.051971435546875, 0.6865234375, -0.6083984375, -0.84423828125, 0.88037109375, 1.623046875, -0.336181640625, 1.01171875, -0.78466796875, 1.1640625, 1.6025390625, -2.234375, -1.3330078125, 0.40966796875, 1.89453125, 2.078125, -0.499267578125, -0.172119140625, -1.74609375, -1.4365234375, 0.265625, -0.27099609375, 1.0263671875, -1.4169921875, -0.8720703125, 1.4921875, -0.50244140625, 1.955078125, -0.81884765625, -0.238037109375, -0.72265625, 0.87109375, -0.25341796875, 0.0908203125, -0.12213134765625, 1.033203125, -0.302978515625, 0.268310546875, -0.36572265625, 0.96337890625, 0.65869140625, -0.3935546875, -0.8505859375, 1.9140625, 2.306640625, 0.84912109375, 1.4775390625, -0.1695556640625, -1.27734375, 1.619140625, -0.12890625, -1.765625, -0.9384765625, -0.024322509765625, 0.445556640625, -0.48828125, 0.10943603515625, 2.546875, -0.91259765625, -0.077880859375, 0.2301025390625, -0.2230224609375, -0.09375, -0.451904296875, 0.89013671875, 1.7978515625, -0.458251953125, -1.4619140625, 0.081298828125, -1.0927734375, -0.3037109375, -0.1192626953125, -0.10174560546875, 0.5634765625, -0.7666015625, 0.47900390625, -0.0206146240234375, -0.9404296875, 0.372802734375, -0.4189453125, 1.5888671875, -0.301513671875, -1.6953125, 0.08489990234375, -0.7802734375, -0.64013671875, 1.1337890625, -1.0458984375, -2.556640625, 0.9287109375, 0.55126953125, -0.1871337890625, -0.50732421875, 1.814453125, -0.62890625, 0.261962890625, -0.859375, -0.1026611328125, -0.2039794921875, -0.300537109375, -1.0625, 0.11920166015625, 1.181640625, 0.8544921875, 0.027099609375, -1.544921875, -0.0660400390625, 1.955078125, -0.1226806640625, -1.0771484375, 1.5830078125, -0.984375, 0.224609375, 0.435791015625, 1.2060546875, -0.81396484375, 0.487548828125, -1.2666015625, 1.6142578125, -0.228271484375, 2.533203125, 0.51123046875, -0.14892578125, 0.94970703125, -0.365234375, -1.4921875, -0.9091796875, -0.5537109375, -0.9580078125, -0.1839599609375, 1.615234375, 1.076171875, -2.197265625, 2.98828125, -0.1326904296875, 1.3779296875, 0.18212890625, -0.84375, -1.4970703125, -0.196533203125, -0.6171875, -0.341796875, 1.810546875, -0.89013671875, 0.4462890625, -0.5068359375, -0.297119140625, 0.830078125, 0.329345703125, -0.3076171875, 0.049102783203125, -1.3642578125, -0.18896484375, 1.1767578125, 0.430419921875, -0.7138671875, -0.144775390625, -0.86328125, 1.107421875, 0.09368896484375, 0.158203125, -0.309326171875, 0.0732421875, 2.166015625, -0.9248046875, -0.28466796875, 0.1011962890625, 0.4384765625, 0.49755859375, 1.84765625, 1.4765625, -0.98388671875, 0.311767578125, 0.0267486572265625, -0.3291015625, -1.3642578125, -0.85107421875, 0.207275390625, -0.3779296875, 1.6240234375, 0.6083984375, -1.2431640625, -0.85009765625, 0.73486328125, 1.29296875, 1.5849609375, 0.666015625, -0.919921875, -1.2763671875, -0.32861328125, -0.8662109375, 0.39453125, 1.537109375, 0.16943359375, -1.07421875, -0.401611328125, 2.9140625, -0.326904296875, 0.7412109375, 0.0009531974792480469, 0.71630859375, 0.8955078125, 1.4638671875, -0.83154296875, 1.435546875, -1.93359375, 0.69384765625, 0.2445068359375, 0.6015625, 1.2900390625, -0.351318359375, -0.87060546875, 0.12841796875, -0.385986328125, 1.0849609375, -0.366943359375, -0.501953125, -0.279052734375, -1.875, -0.21728515625, -1.7685546875, 1.255859375, -0.0248260498046875, -0.7783203125, 0.292724609375, -0.332275390625, 0.09295654296875, -0.495849609375, 0.18505859375, 0.481201171875, -0.685546875, -0.61474609375, 2.052734375, 0.4755859375, 1.4072265625, -0.6884765625, -0.05609130859375, -0.8076171875, 1.578125, 0.599609375, -0.1387939453125, 2.126953125, 0.0242919921875, -0.76220703125, -1.734375, -0.91064453125, 0.71142578125, 0.96240234375, 0.09246826171875, -0.413330078125, 3.5, -0.4599609375, 0.1580810546875, 0.387451171875, 0.62890625, 0.9033203125, 0.2392578125, 1.294921875, -0.1907958984375, 0.78759765625, -0.61376953125, -0.89501953125, -0.1480712890625, -1.271484375, 0.43994140625, 1.5166015625, 1.7265625, 0.11273193359375, 0.2119140625, 1.6806640625, 1.9482421875, -0.27099609375, 0.34814453125, -0.49951171875, -1.5078125, -1.541015625, 0.041290283203125, 1.7236328125, -0.11083984375, 0.91796875, -0.178466796875, -1.3701171875, 2.029296875, -0.61669921875, 1.0234375, 0.62548828125, -1.193359375, -1.390625, -2.08984375, -0.09185791015625, -1.25, 0.026458740234375, 1.703125, -0.6484375, -0.7802734375, -0.6025390625, -0.53271484375, -0.6416015625, 0.90380859375, 0.912109375, 0.7646484375, -0.689453125, 0.837890625, 0.6552734375, -0.60205078125, 0.224853515625, 1.41015625, -0.321533203125, -0.8125, -1.47265625, -0.1318359375, -0.75439453125, -0.89453125, 0.85498046875, 0.98876953125, 0.6728515625, 0.7060546875, 0.39013671875, -0.398681640625, 0.73583984375, -0.473388671875, 0.0293121337890625, -0.277587890625, 1.4580078125, -0.2393798828125, 0.211181640625, 0.82763671875, 0.438232421875, 0.3671875, -0.27392578125, 0.6181640625, -1.490234375, -0.09564208984375, -0.36328125, 0.5849609375, -0.1795654296875, -0.71630859375, -0.9931640625, -0.159423828125, -1.6474609375, -0.19140625, -0.6298828125, -1.6962890625, 1.642578125, 0.376708984375, 0.449462890625, 0.25341796875, 1.5400390625, -0.35986328125, 0.1871337890625, 0.7705078125, 0.958984375, -0.477294921875, 0.05169677734375, 2.189453125, -1.19921875, 0.89697265625, -0.491943359375, 0.39306640625, 0.5302734375, -0.45263671875, 0.9501953125, 0.82861328125, -0.7998046875, -1.5556640625, 0.69580078125, 0.76904296875, -1.3330078125, 0.348388671875, 1.5166015625, 0.204345703125, 1.025390625, -1.2705078125, -0.4599609375, 0.08795166015625, 1.1044921875, -0.243896484375, 0.23193359375, 0.50048828125, -2.21875, 0.57666015625, 0.62451171875, 1.490234375, 0.2291259765625, -0.05615234375, 0.52294921875, 0.436767578125, -1.4814453125, 0.76025390625, -0.65771484375, -2.123046875, 0.1297607421875, -0.6552734375, 0.436767578125, 0.4775390625, -0.58349609375, 0.748046875, 1.474609375, 0.73876953125, 0.77490234375, 0.93212890625, -2.03515625, -0.94580078125, -0.873046875, -1.314453125, -0.34814453125, -0.52734375, -0.533203125, -1.3115234375, 0.47265625, -0.8046875, -0.7216796875, 0.0099029541015625, -0.20703125, 0.37451171875, -0.353759765625, 0.487060546875, 0.0233306884765625, 0.327880859375, 0.261474609375, 1.7421875, -1.1337890625, -0.9658203125, 0.76513671875, 0.5576171875, 0.0300140380859375, 1.4794921875, -1.7099609375, -0.416015625, -0.76025390625, 0.77490234375, -2.142578125, -0.7333984375, -1.61328125, 1.173828125, 0.64697265625, -0.061248779296875, 1.712890625, 1.3095703125, 1.0400390625, -0.5107421875, -0.826171875, 1.69140625, 1.8173828125, -0.6220703125, -1.322265625, -0.65576171875, 1.234375, 0.24267578125, -1.1005859375, 1.310546875, 0.1866455078125, 0.197021484375, -0.80078125, -0.49951171875, -0.1019287109375, -0.335205078125, 1.6611328125, 0.984375, -0.6708984375, -0.7431640625, -1.6669921875, -0.412353515625, 0.37109375, -1.2978515625, 0.038116455078125, -1.54296875, -1.5634765625, 0.7900390625, -0.70361328125, -0.6552734375, 0.5244140625, -1.6455078125, -0.9033203125, -1.083984375, 1.1328125, 2.3671875, 1.341796875, -0.2137451171875, 0.2149658203125, -0.040283203125, 0.0021495819091796875, 1.931640625, -1.166015625, 0.5390625, 0.08111572265625, -0.72314453125, 0.86474609375, -1.138671875, 1.2236328125, -1.22265625, -1.69140625, -1.640625, -0.250732421875, 1.5009765625, 0.410400390625, -0.24365234375, 0.619140625, -1.087890625, -0.2413330078125, -0.86572265625, 0.75732421875, -0.55517578125, -1.642578125, 0.87060546875, 0.38427734375, 0.320556640625, -1.01171875, -0.693359375, -0.52734375, 0.63525390625, -0.5751953125, -0.89208984375, 0.8583984375, 0.319580078125, 0.53759765625, 0.485107421875, -0.85400390625, -0.91015625, 0.22412109375, -0.311767578125, 0.8154296875, 0.419921875, -1.0048828125, -1.3388671875, 0.59033203125, -0.35107421875, 0.54833984375, 0.34033203125, 0.78564453125, 0.133544921875, 0.55419921875, -0.39453125, 0.346435546875, -1.9501953125, 0.13525390625, 2.203125, -1.01171875, -0.63916015625, -1.859375, -1.767578125, 0.67333984375, 0.85498046875, -0.48876953125, -0.1361083984375, -0.94775390625, 0.0767822265625, 1.6845703125, 0.258056640625, -0.83935546875, -2.1796875, -2.119140625, -0.83251953125, 1.498046875, 0.325439453125, 0.63818359375, -0.148681640625, 0.88818359375, -1.15625, -0.2467041015625, 0.9130859375, -1.3115234375, 1.21875, -0.050201416015625, 1.466796875, 0.7451171875, -0.2386474609375, -0.0030689239501953125, -1.1220703125, 1.7255859375, 1.8828125, -0.63818359375, -0.234130859375, 1.912109375, -1.615234375, -1.1328125, 0.2734375, -2.7734375, 1.2939453125, 0.48193359375, 1.0712890625, 0.03857421875, -2.404296875, 0.492431640625, 0.36865234375, -0.37841796875, -0.438720703125, 1.482421875, -0.4921875, -0.1744384765625, -0.2191162109375, -0.445556640625, 0.050567626953125, -0.52001953125, -1.2099609375, -0.5322265625, 0.0029239654541015625, -0.83056640625, 1.046875, -0.40673828125, -0.50927734375, -0.485107421875, -0.7568359375, -0.0811767578125, -0.297119140625, -0.53466796875, 1.0654296875, -0.66943359375, 0.34716796875, -0.30908203125, 0.77001953125, 0.5869140625, -0.00501251220703125, 0.244384765625, -0.1968994140625, -1.287109375, -0.544921875, -0.00896453857421875, 0.822265625, -0.85009765625, 0.0640869140625, -0.9794921875, 0.187255859375, 1.8671875, 1.0810546875, -2.44921875, -0.83837890625, 1.6201171875, -0.783203125, -0.70361328125, 1.919921875, 1.013671875, -0.15625, -0.37646484375, -1.5068359375, 1.69140625, 1.80859375, 0.873046875, -0.1961669921875, -0.08050537109375, 0.58837890625, -0.57470703125, -0.4970703125, -0.05902099609375, 0.30029296875, 0.10687255859375, -0.6025390625, -1.9140625, 0.60009765625, -1.591796875, 0.77392578125, 0.014373779296875, -0.30517578125, -0.46337890625, -1.0078125, -0.8525390625, -0.040496826171875, -0.70849609375, 1.5400390625, 0.525390625, 0.8251953125, -1.48046875, -0.09869384765625, 0.78662109375, -1.2373046875, 0.66357421875, 0.52685546875, -1.689453125, -1.7958984375, -1.033203125, 0.68359375, 0.5732421875, -2.119140625, -2.974609375, 0.0450439453125, 0.378662109375, -1.259765625, 0.3125, -0.92138671875, 0.81591796875, 1.337890625, 0.73583984375, 0.86376953125, 0.77392578125, -0.67578125, -0.224609375, -0.9033203125, -1.5908203125, 1.5146484375, 0.258544921875, -0.58447265625, -1.9033203125, -0.86279296875, 0.1416015625, 1.0283203125, -0.419677734375, -1.1328125, -0.127197265625, -0.27392578125, 1.1220703125, 0.60498046875, -0.5087890625, -0.1827392578125, 1.0380859375, 1.51953125, -1.6201171875, 0.1392822265625, -0.51806640625, -0.234130859375, 0.998046875, -0.84326171875, -0.15625, -1.1328125, -1.708984375, -0.153076171875, -0.90087890625, 1.462890625, 0.143798828125, 0.25830078125, 0.90673828125, -0.66845703125, -1.193359375, 1.3896484375, 1.849609375, 0.420166015625, -0.955078125, 1.1240234375, -0.0016050338745117188, 0.27783203125, 0.5771484375, -0.1405029296875, -0.1990966796875, 1.283203125, -1.1103515625, -0.5615234375, 0.415283203125, -0.7861328125, 0.371826171875, -0.58544921875, 1.4677734375, -1.416015625, -0.767578125, -0.6337890625, 0.11090087890625, -0.11199951171875, -0.93994140625, 0.054290771484375, -1.5419921875, -0.0254364013671875, -0.0229034423828125, 1.32421875, -0.56884765625, 0.12091064453125, -0.82421875, 0.640625, -1.765625, -0.316162109375, -0.96533203125, 1.9130859375, 0.47802734375, -0.53466796875, 0.99755859375, 0.8671875, -0.208740234375, 1.2783203125, -0.86376953125, 0.298583984375, -0.53857421875, -0.615234375, -0.66015625, -0.214111328125, 0.91015625, -0.1431884765625, -2.45703125, 0.1553955078125, 0.8251953125, 0.34326171875, 0.2432861328125, -0.29052734375, 1.3056640625, -0.223388671875, -0.52734375, 1.197265625, 1.4970703125, 0.3642578125, 0.156494140625, 0.2164306640625, 0.5888671875, 1.7197265625, 1.046875, -0.1263427734375, -2.17578125, 0.254150390625, 0.391845703125, -1.0478515625, 0.2802734375, -0.7880859375, 1.7900390625, 0.2076416015625, -1.330078125, 0.7587890625, -0.007099151611328125, -1.0322265625, 0.60302734375, -1.3486328125, 0.78466796875, 0.7060546875, 1.7568359375, 1.3125, 1.384765625, 0.53955078125, 0.07989501953125, -0.19287109375, 0.40869140625, -0.415771484375, -0.445556640625, 2.14453125, -0.67529296875, -0.697265625, 0.092041015625, -0.61474609375, 1.513671875, 0.10382080078125, 0.3583984375, 0.7333984375, 1.0078125, -0.444580078125, -0.1397705078125, 0.227783203125, 0.466796875, -1.5283203125, 0.313232421875, -1.3603515625, 0.284423828125, -1.2587890625, -0.572265625, -2.220703125, -1.5859375, -1.111328125, 0.6982421875, -1.7158203125, -0.2171630859375, -0.8837890625, 0.61376953125, 2.80078125, 0.25634765625, 0.79638671875, 0.218994140625, -1.5234375, 1.04296875, 0.58203125, 0.58935546875, 1.078125, -1.3466796875, 1.0908203125, -1.4794921875, -1.2451171875, 0.483154296875, 1.150390625, 0.281982421875, 1.787109375, -0.51123046875, 1.3173828125, 0.8564453125, 1.2529296875, -0.099365234375, 0.63671875, 0.73583984375, 1.751953125, 0.161865234375, 0.2369384765625, -0.58642578125, -0.347412109375, -0.5830078125, 0.04833984375, 0.55810546875, 0.97314453125, -2.96875, 0.96240234375, -0.2296142578125, 0.61328125, -0.57470703125, 1.2060546875, -1.958984375, 0.15966796875, 0.7802734375, -1.23828125, 1.0732421875, -1.3154296875, 2.55859375, 1.2431640625, 0.1171875, -0.1270751953125, 0.44189453125, 0.21533203125, -0.091552734375, 0.08099365234375, -1.271484375, 0.1920166015625, 0.6572265625, -0.3154296875, -0.218017578125, 0.156005859375, -0.56298828125, 1.236328125, -1.775390625, 1.1669921875, 0.38623046875, 0.156982421875, 0.485107421875, -2.326171875, -0.399169921875, 0.407958984375, 0.4677734375, 0.58544921875, -0.31201171875, -1.392578125, 0.2464599609375, -0.34033203125, -1.5498046875, 0.02996826171875, -0.73779296875, -1.0791015625, -0.56494140625, -0.91162109375, 0.369140625, 0.33837890625, 2.345703125, 0.5263671875, 0.3662109375, -0.73486328125, -2.529296875, 0.8603515625, -0.1751708984375, 0.6640625, 0.58837890625, -1.728515625, -0.446044921875, -1.080078125, 0.14892578125, -2.50390625, -0.84423828125, 1.056640625, -0.2197265625, 0.75390625, 0.94873046875, 0.39208984375, 1.4716796875, -0.2060546875, -2.068359375, 0.143798828125, -0.6025390625, -0.4794921875, -0.268798828125, 0.22607421875, 0.429443359375, 1.189453125, -0.6103515625, 0.02349853515625, 1.5810546875, -0.30224609375, 1.32421875, 1.1865234375, 1.3173828125, -0.52734375, 1.5029296875, -1.4599609375, 0.0430908203125, 0.65625, 0.3466796875, 1.4658203125, -1.17578125, -1.6904296875, 0.3740234375, -0.151123046875, 1.341796875, 1.6298828125, 0.947265625, -0.11822509765625, 0.6474609375, 1.142578125, 0.70166015625, -0.74755859375, -0.058074951171875, 1.001953125, 1.3388671875, 0.7939453125, -0.085693359375, -0.0467529296875, -1.1630859375, 0.67333984375, 0.149658203125, 0.348876953125, -0.4423828125, -0.32470703125, 0.0224761962890625, -0.61181640625, -1.544921875, 0.469970703125, -0.20947265625, -0.6865234375, 0.83251953125, 0.73095703125, -1.0068359375, 1.673828125, -0.58544921875, -0.228759765625, 1.19140625, -0.47705078125, -1.1787109375, 0.20166015625, -0.3125, 2.0234375, 0.5517578125, -1.1416015625, -1.40625, 2.267578125, 1.044921875, 0.52197265625, -0.36669921875, 0.69580078125, -1.2646484375, -1.3662109375, 0.01519775390625, 0.3154296875, -0.8515625, 1.0224609375, -1.7060546875, -1.1455078125, 0.257080078125, 1.005859375, -1.4853515625, -0.720703125, -0.149169921875, -0.40576171875, -0.405517578125, -0.69287109375, 0.58056640625, 0.98486328125, 1.8125, 0.857421875, 0.42138671875, -1.1142578125, 1.0888671875, 0.1072998046875, -0.449462890625, 1.8662109375, 0.1119384765625, -0.148193359375, -0.29150390625, 0.8857421875, -1.361328125, -2.7890625, 0.96728515625, 0.11517333984375, -1.1630859375, -0.302978515625, 1.0419921875, 0.96875, 0.1246337890625, -0.6376953125, 0.18212890625, 1.37890625, 1.169921875, -0.43994140625, 0.1468505859375, -1.2802734375, 0.69091796875, 0.1512451171875, 2.037109375, 0.7119140625, -2.244140625, 1.4248046875, 1.33984375, 0.80126953125, 0.1708984375, -0.2479248046875, -0.321533203125, 1.6328125, 1.240234375, 0.068603515625, -0.615234375, -2.015625, -1.693359375, -0.26171875, -1.2509765625, 0.625, 0.63232421875, -0.349609375, 1.935546875, -0.0953369140625, -2.365234375, -0.361083984375, -0.057525634765625, 0.69921875, 1.580078125, 0.8955078125, 0.65625, -0.9482421875, 1.322265625, -1.2490234375, -1.390625, -0.093017578125, 0.29150390625, 0.8623046875, -0.53564453125, -1.048828125, 1.8896484375, 0.83203125, -1.037109375, 2.46875, 1.9365234375, -1.7119140625, 1.802734375, -2.5, -0.54638671875, -1.6533203125, 0.52783203125, -0.113037109375, -0.265869140625, 0.219482421875, -0.9677734375, 0.94091796875, -0.254638671875, 1.0791015625, 0.66064453125, 0.1800537109375, -0.7783203125, -1.85546875, 0.1220703125, -0.74853515625, -0.1827392578125, -0.4794921875, 1.6416015625, 0.042938232421875, -0.7841796875, 1.51171875, 0.127197265625, 0.35791015625, -1.3525390625, 1.478515625, -1.375, 0.6591796875, -1.423828125, 0.361083984375, 0.03460693359375, -0.1107177734375, -0.8154296875, 1.9677734375, -0.7099609375, 0.0007381439208984375, -0.45068359375, 1.5595703125, 1.185546875, 0.611328125, 1.7822265625, -1.103515625, 0.341552734375, 1.03515625, -0.383056640625, -0.087890625, -0.62890625, 1.4091796875, 1.4072265625, 1.923828125, 0.455078125, -1.033203125, 1.8720703125, 0.383544921875, 0.476318359375, -0.306396484375, 0.33544921875, 0.8505859375, -0.37744140625, -1.322265625, 0.69970703125, 0.2398681640625, -0.3876953125, 0.370849609375, -0.12066650390625, 0.285888671875, -0.14599609375, -0.373291015625, -0.3544921875, -1.232421875, -0.36865234375, 1.71875, 1.2021484375, 0.212158203125, -0.2265625, 0.142822265625, 1.4990234375, 1.515625, -0.398193359375, -1.4111328125, -0.63671875, -0.07928466796875, -0.80517578125, -0.60302734375, -0.2471923828125, -1.052734375, -0.67529296875, -0.65673828125, -0.7119140625, -2.267578125, -0.053253173828125, 0.8359375, 1.4775390625, -1.3642578125, 1.4921875, 1.2421875, 0.65283203125, 0.509765625, -1.3349609375, 0.74609375, -0.2171630859375, -2.845703125, -1.1259765625, -0.80810546875, 1.669921875, 0.28564453125, -0.787109375, 1.033203125, -1.458984375, -1.69140625, -1.357421875, 0.1947021484375, -1.140625, 0.2783203125, -0.00800323486328125, 0.201904296875, -0.611328125, -0.9423828125, 1.2060546875, 0.2076416015625, -1.564453125, 0.6376953125, 1.9072265625, 0.939453125, -1.0517578125, -1.3232421875, 0.5703125, 0.904296875, 0.615234375, -0.2890625, 0.2266845703125, -0.55859375, 0.377685546875, 1.5771484375, 0.294921875, 1.0576171875, 0.2451171875, 0.48193359375, 0.04010009765625, 0.051910400390625, -0.91650390625, 0.42236328125, 1.1142578125, -0.414794921875, 1.2744140625, -0.51904296875, 0.326904296875, -2.564453125, 0.45947265625, 1.1025390625, 0.38525390625, 0.7021484375, -1.20703125, 0.662109375, 3.94921875, 1.6435546875, 1.1015625, -0.483642578125, 0.95654296875, 1.611328125, 1.083984375, 0.90087890625, -0.325927734375, -2.923828125, 0.013946533203125, -0.91162109375, 2.13671875, -1.11328125, -0.1336669921875, 0.1676025390625, -0.72021484375, -1.3212890625, 1.3017578125, -0.271728515625, -0.72705078125, 0.51611328125, 1.0927734375, -0.76708984375, 0.0008182525634765625, -0.2083740234375, 0.55078125, -1.4755859375, 0.2230224609375, -0.68896484375, -0.191650390625, 1.2763671875, -1.4814453125, 0.443115234375, 0.1629638671875, -2.01171875, 0.364501953125, -1.2919921875, -0.3525390625, -1.455078125, -0.7880859375, 0.5927734375, -1.5390625, -0.178466796875, -0.2125244140625, 0.212890625, 0.34228515625, -2.115234375, 0.4091796875, -2.525390625, -0.184814453125, -0.302734375, 0.266357421875, -0.035919189453125, -0.87060546875, -0.537109375, -2.1875, -0.84814453125, 1.6083984375, -0.386962890625, -0.172119140625, -0.57666015625, 1.017578125, 0.204345703125, -1.013671875, -1.384765625, 0.305908203125, -0.386474609375, -0.410400390625, 0.02679443359375, 1.6826171875, 0.330078125, 0.4296875, 0.7568359375, 0.478271484375, -0.837890625, -0.266845703125, 0.8115234375, -1.0009765625, -0.2080078125, -1.7138671875, 0.67236328125, -0.7177734375, -0.896484375, -0.1427001953125, 1.0517578125, -0.057373046875, 0.6884765625, -1.2587890625, -0.7529296875, -0.9814453125, -0.48583984375, -0.662109375, -0.1881103515625, 0.8232421875, 0.73779296875, 0.1453857421875, -0.470458984375, 0.2340087890625, -0.47314453125, 0.72509765625, -1.08203125, 0.235107421875, 1.5888671875, 0.62353515625, 0.295166015625, 0.7421875, -0.6396484375, 0.3818359375, 0.751953125, -1.6435546875, -2.6953125, 0.0872802734375, 0.671875, -0.92529296875, 0.978515625, -1.123046875, 2.076171875, 0.56787109375, -0.55517578125, 1.30859375, 0.329833984375, 1.4189453125, 0.96923828125, 0.0231475830078125, -0.12078857421875, 1.4873046875, -1.2021484375, -1.482421875, -1.5087890625, 1.0322265625, 1.2685546875, -0.5439453125, -0.25341796875, 2.759765625, -1.5908203125, 0.06561279296875, 2.2734375, 1.41015625, -0.2418212890625, 1.005859375, -0.6552734375, 0.51611328125, 1.54296875, -0.06817626953125, -0.77587890625, 0.77001953125, 2.61328125, -0.1614990234375, 0.8369140625, 1.2275390625, 0.376220703125, -1.109375, 0.051116943359375, -0.8779296875, 2.451171875, -0.1650390625, -0.58837890625, -0.06231689453125, -1.478515625, 1.48828125, 1.1162109375, 0.57470703125, -1.1826171875, -1.1142578125, 0.322265625, 1.4892578125, -0.7314453125, -0.62939453125, -0.1480712890625, -1.8916015625, -1.587890625, -0.91064453125, -1.0849609375, -1.0400390625, -0.153076171875, -0.474365234375, -0.12274169921875, 1.05078125, 0.54833984375, -1.185546875, -0.64599609375, -0.169189453125, 1.3974609375, -1.701171875, 1.49609375, 1.2353515625, -0.1290283203125, -1.3974609375, 0.349609375, 0.30810546875, 0.89697265625, -0.2147216796875, -2.16015625, 0.36181640625, 0.5224609375, -1.5478515625, -0.7197265625, 0.79248046875, -1.19140625, -1.3310546875, 0.57080078125, 0.8984375, -1.068359375, -0.45458984375, -1.5166015625, 0.15185546875, 1.0703125, 0.9501953125, 0.6640625, 1.8974609375, -2.228515625, -0.1666259765625, 0.42138671875, 0.2200927734375, 0.65087890625, -1.1845703125, 1.1025390625, 1.2578125, -0.1329345703125, -0.51953125, 0.97509765625, 1.0029296875, 0.7236328125, -0.236572265625, -1.009765625, -1.974609375, -1.34765625, 0.45654296875, -0.041748046875, -0.2264404296875, 0.367919921875, -0.89453125, 0.82861328125, 1.6220703125, -0.67138671875, -2.017578125, -1.0810546875, 1.7939453125, 0.376953125, -0.51904296875, -0.1519775390625, 0.129150390625, 1.1044921875, 0.54296875, 0.2274169921875, -0.4169921875, -1.14453125, 0.580078125, 1.3056640625, -0.775390625, -0.0009241104125976562, 0.237548828125, 0.10284423828125, 2.48046875, -1.0302734375, 1.4072265625, -0.042755126953125, 0.96435546875, -1.3623046875, 0.0032825469970703125, -2.1015625, 0.08831787109375, 0.56494140625, -0.369873046875, 0.6533203125, 0.8759765625, -1.1806640625, -0.2232666015625, 0.4619140625, -0.1898193359375, -0.7236328125, -0.93505859375, 0.98583984375, 0.802734375, 0.82275390625, -0.64453125, -0.1470947265625, 0.6259765625, -1.2392578125, 0.5634765625, 1.5986328125, -0.42529296875, -1.6015625, 0.8818359375, -0.9560546875, -0.7080078125, -0.30517578125, 1.0888671875, -0.80029296875, -0.52685546875, -1.166015625, 0.291259765625, 0.196044921875, -0.87841796875, -0.67041015625, 1.064453125, -0.0303497314453125, -1.2080078125, 1.0537109375, 0.53515625, -1.259765625, 0.69189453125, -0.228271484375, -0.287353515625, -0.77294921875, -1.82421875, 0.8935546875, 0.02166748046875, 0.1695556640625, -0.953125, 1.4833984375, -0.341796875, -0.91650390625, 0.1610107421875, 0.5400390625, -1.0546875, 0.033203125, -0.74267578125, -1.0126953125, -0.290283203125, 0.53662109375, -0.1494140625, -0.52783203125, -1.55078125, -1.890625, -1.740234375, -1.3330078125, -1.5859375, 0.87158203125, -0.2431640625, 0.0521240234375, 1.3037109375, 1.4990234375, -0.74169921875, -0.037384033203125, 0.37109375, -0.513671875, -2.083984375, 1.0029296875, -0.304443359375, 0.1387939453125, 0.642578125, 0.92919921875, 1.8623046875, -0.64306640625, -0.8955078125, -0.85888671875, 0.20751953125, 0.45703125, -0.552734375, -0.9375, 1.4716796875, 0.11383056640625, 0.1053466796875, 0.35009765625, -2.15625, -1.8388671875, -1.0830078125, 0.220947265625, 2.6328125, -2.025390625, 0.250732421875, 0.79248046875, 1.205078125, 0.548828125, -1.216796875, 0.34423828125, -0.70751953125, 0.061248779296875, -0.255615234375, 0.27099609375, -0.406005859375, 0.435791015625, 0.028533935546875, 0.568359375, -0.94677734375, -0.30126953125, -0.288330078125, -0.92333984375, 0.4775390625, -1.6044921875, 0.9794921875, -0.06512451171875, 0.494873046875, 0.548828125, -0.51708984375, -1.4345703125, 1.619140625, -0.58251953125, -0.320068359375, -0.32275390625, -3.197265625, 1.19921875, 0.6337890625, 2.208984375, -0.7265625, 0.462890625, -1.0654296875, 0.299072265625, -0.166259765625, -1.5595703125, 0.54931640625, -0.1246337890625, -0.343017578125, -0.88232421875, -1.3193359375, 1.291015625, 0.220458984375, 0.3125, 1.5400390625, -1.5908203125, -1.634765625, -1.0634765625, 1.52734375, -0.60107421875, 0.6474609375, -0.5546875, -0.2125244140625, 0.392578125, 0.267578125, 0.18212890625, -0.10577392578125, 0.1417236328125, 0.99658203125, -1.103515625, 0.2396240234375, -0.7763671875, -0.91552734375, 1.0849609375, -0.6337890625, 0.52490234375, 0.77294921875, 0.4892578125, -0.96923828125, 2.23046875, -1.69921875, 1.0634765625, -1.4521484375, 0.30126953125, 1.923828125, 0.9111328125, -0.9052734375, -0.28125, 1.51953125, 0.442138671875, -0.51123046875, 0.277099609375, 0.434814453125, -0.8017578125, 0.25, 0.27587890625, 1.6123046875, -1.025390625, 0.865234375, -1.34375, -0.716796875, 0.0897216796875, 0.8408203125, 0.83349609375, -0.27197265625, 1.4677734375, 0.2900390625, -2.673828125, 0.65966796875, 0.3017578125, -0.27197265625, 0.02984619140625, 1.640625, -0.11138916015625, -0.415283203125, 0.96923828125, 0.209228515625, -2.05859375, 0.76025390625, 1.4970703125, -0.6376953125, 0.0203704833984375, 0.85791015625, -2.59375, -0.33544921875, 1.037109375, 1.244140625, -2.236328125, -0.295166015625, 0.5654296875, -1.13671875, -3.708984375, -0.275146484375, 0.50537109375, 2.01953125, 2.6171875, -0.049346923828125, -0.52685546875, 0.66650390625, 0.427978515625, 0.443115234375, -0.1278076171875, -0.1553955078125, 0.525390625, 0.55517578125, 0.71337890625, -1.755859375, 0.83251953125, -1.0078125, -1.8662109375, 2.34375, 1.09765625, -0.262939453125, -0.9814453125, 0.09844970703125, 0.54248046875, -0.357421875, -1.0546875, -0.6923828125, -0.1551513671875, -0.489501953125, 0.56591796875, 0.269287109375, 0.8876953125, -0.6767578125, 0.82421875, 0.0380859375, -0.199462890625, -1.1044921875, -0.1612548828125, 0.681640625, -1.208984375, 0.974609375, 0.7333984375, 0.63720703125, -0.38427734375, 0.271728515625, -0.533203125, -0.72265625, 0.62548828125, 0.5087890625, -0.89306640625, -0.79443359375, -0.35595703125, -1.1806640625, -0.25439453125, 0.062255859375, 0.25927734375, -0.453369140625, -0.576171875, -0.1517333984375, -0.464599609375, -0.346923828125, -0.492919921875, 0.372802734375, -0.2254638671875, 1.3359375, -0.4521484375, 0.56787109375, -2.09375, -0.432373046875, -0.12353515625, 1.8125, 0.5146484375, 0.87841796875, 1.6005859375, 0.009735107421875, -0.400390625, -0.61767578125, -0.444580078125, -0.38037109375, -1.67578125, 0.007488250732421875, -1.595703125, -0.7978515625, -0.5595703125, 1.287109375, 1.7802734375, 0.2265625, 0.448974609375, -0.75146484375, -1.32421875, -0.283203125, 0.1717529296875, -1.595703125, 0.056793212890625, 1.44921875, -0.57861328125, -0.2081298828125, -0.66357421875, -0.36328125, -0.1175537109375, -1.5341796875, -0.40234375, -0.35693359375, 0.2431640625, 0.7001953125, -1.4892578125, -0.337890625, -0.97021484375, -0.2900390625, 0.22705078125, -0.2425537109375, -0.65283203125, 1.2568359375, 1.0341796875, -0.857421875, 0.1507568359375, 0.53271484375, 0.9716796875, -0.055145263671875, 0.9873046875, 0.64501953125, -0.97900390625, -0.20556640625, 0.10528564453125, 0.935546875, 1.134765625, -0.2147216796875, 1.72265625, -0.943359375, 0.1009521484375, 0.3232421875, -1.1806640625, 0.30859375, -2.43359375, 0.3408203125, 1.0400390625, -0.475341796875, -0.787109375, -0.489990234375, -1.3154296875, -2.12109375, 1.9326171875, 0.2220458984375, -0.955078125, 0.880859375, 0.74951171875, 0.040863037109375, -0.306640625, 1.14453125, -0.21630859375, 0.044586181640625, 1.638671875, -1.3427734375, -0.7958984375, -0.377197265625, -1.2841796875, -0.60498046875, -1.544921875, 0.277587890625, 0.34521484375, 1.142578125, -0.218505859375, 0.4921875, 0.129150390625, 0.99169921875, 0.43701171875, -0.7255859375, -0.1351318359375, 0.7958984375, 0.487548828125, -0.326171875, 0.436767578125, -0.325439453125, 1.34375, -0.8759765625, 0.85791015625, 0.1624755859375, 0.08111572265625, 0.40234375, 0.5390625, 0.0963134765625, -0.059326171875, 0.35107421875, 0.05322265625, 0.67041015625, 0.99267578125, -0.290283203125, -0.0293426513671875, -0.0265045166015625, 0.47119140625, 0.8759765625, -0.95947265625, -0.560546875, -0.8173828125, 1.123046875, 0.477783203125, 1.6982421875, 0.9189453125, 1.33984375, -0.4765625, 1.087890625, 0.59716796875, -1.4521484375, 1.16015625, -0.630859375, -0.67822265625, 0.82080078125, 1.69921875, -0.1939697265625, 0.24072265625, -0.134765625, 1.2255859375, 0.2406005859375, -0.751953125, -0.273681640625, 2.11328125, -0.314208984375, -1.8251953125, -0.265380859375, 0.23779296875, 1.46484375, 0.7802734375, -0.52001953125, 0.258056640625, 0.521484375, -1.1416015625, 0.06689453125, -0.18017578125, -0.055389404296875, -0.427734375, 0.5654296875, -0.3935546875, -0.023468017578125, 1.6201171875, 0.183837890625, 0.5205078125, -1.0546875, -0.07525634765625, -0.37548828125, 0.61474609375, 0.448486328125, 1.3251953125, -1.099609375, 3.572265625, 0.96044921875, 0.828125, 0.93701171875, -0.5419921875, -0.59814453125, -0.49853515625, 2.3359375, -0.2451171875, 0.391845703125, -0.353515625, 1.2548828125, 0.51416015625, -1.248046875, 0.87353515625, -0.347900390625, 0.87255859375, 0.6455078125, 2.61328125, 0.303955078125, 0.42041015625, 1.177734375, -1.0927734375, 0.54345703125, -0.5029296875, -0.391357421875, -1.6416015625, 0.1136474609375, -0.54736328125, 1.6123046875, -0.244873046875, -1.0947265625, 1.1884765625, -1.6552734375, 1.041015625, -0.439208984375, -2.439453125, -0.68896484375, 1.634765625, 1.5458984375, 1.1103515625, 0.515625, -1.2939453125, 0.35205078125, 1.7841796875, 0.52734375, 1.0009765625, 2.267578125, 0.2222900390625, -1.1171875, 0.26171875, -2.125, -1.587890625, -1.828125, -1.392578125, 0.82568359375, 0.92431640625, 0.8095703125, -1.013671875, 0.1480712890625, 0.041473388671875, -0.211181640625, 0.439697265625, -0.2095947265625, 0.097900390625, -1.17578125, -0.1573486328125, 2.830078125, -0.006099700927734375, 0.947265625, 0.85498046875, 0.006320953369140625, 1.25, 2.662109375, 1.2265625, -0.098876953125, 0.11676025390625, 0.331298828125, -1.23828125, -1.68359375, -0.058685302734375, -0.224609375, -0.06951904296875, 0.83740234375, 0.1944580078125, -1.341796875, -0.36083984375, -0.1328125, -0.23583984375, -1.2509765625, -0.102294921875, 1.296875, 2.232421875, -1.1435546875, -0.654296875, -0.0013828277587890625, -0.296875, 0.042236328125, 0.392333984375, 0.307861328125, 0.787109375, 0.09210205078125, -1.8369140625, 0.75244140625, 1.4658203125, -0.75927734375, 0.58984375, 1.06640625, -0.67431640625, -0.98193359375, 0.6025390625, -0.734375, 0.02734375, -0.46337890625, -0.87841796875, -0.8818359375, 1.943359375, 0.62158203125, 0.75439453125, -0.888671875, 1.6337890625, -0.6396484375, -0.8525390625, 0.86474609375, -0.4814453125, -0.2197265625, -0.10498046875, 0.8173828125, -0.327392578125, 0.11871337890625, -1.1513671875, 0.496826171875, -0.56640625, 0.3076171875, 0.7158203125, 0.7197265625, -0.798828125, 0.783203125, 0.07781982421875, -0.296142578125, -0.84326171875, 0.97705078125, -0.7880859375, 0.58203125, 0.5615234375, -1.2236328125, -1.771484375, 0.106689453125, 0.46044921875, 1.41796875, 0.58056640625, -0.32373046875, 0.4150390625, 0.826171875, -0.436279296875, -0.2088623046875, 1.01171875, -0.04150390625, 0.654296875, -0.1297607421875, 0.47412109375, -1.298828125, 0.57421875, -0.8603515625, 0.2305908203125, 1.01953125, -0.09185791015625, 0.006084442138671875, 1.2783203125, 0.5283203125, 0.315673828125, -0.01032257080078125, 1.73046875, 0.87939453125, 0.939453125, 0.89990234375, 0.9765625, -0.412109375, 2.15234375, 1.5927734375, 1.6396484375, -0.10736083984375, -1.814453125, 2.037109375, -0.966796875, -1.4990234375, -1.708984375, 0.6015625, -0.043548583984375, 0.853515625, 1.6240234375, -0.0943603515625, 1.439453125, 0.83056640625, 0.8115234375, 0.271728515625, -0.26513671875, 0.297607421875, 0.740234375, -2.349609375, 0.221435546875, 0.5, -0.63818359375, -1.6455078125, -1.8720703125, -1.3056640625, -1.501953125, -1.0888671875, 1.197265625, -0.71630859375, 1.9775390625, 0.06744384765625, 0.517578125, 0.18994140625, -0.317626953125, -1.97265625, 0.460205078125, 1.2109375, 1.443359375, -1.33203125, -1.4404296875, 0.58154296875, 1.0029296875, -0.73779296875, -1.193359375, 0.41943359375, 0.892578125, -0.91796875, 0.123779296875, 0.492919921875, 0.22412109375, 2.423828125, 0.2144775390625, 1.904296875, 1.876953125, -0.75732421875, -0.244384765625, -0.67138671875, 0.59912109375, -0.0406494140625, -0.12744140625, -1.27734375, -1.5419921875, -0.76220703125, 0.44921875, -1.11328125, 0.08563232421875, 0.67919921875, 0.1845703125, 2.26953125, 2.8984375, -2.142578125, -0.362060546875, 0.0199432373046875, 0.7568359375, 0.6865234375, 0.77978515625, -0.21337890625, -0.54638671875, -0.5341796875, 0.75146484375, 0.18603515625, 0.10589599609375, 0.446533203125, 0.8193359375, 0.09210205078125, -0.9697265625, -0.8642578125, -1.984375, -0.201416015625, 2.412109375, -0.52392578125, 0.9951171875, -0.6669921875, -1.0400390625, 1.052734375, 0.7451171875, -0.57080078125, -0.1114501953125, 0.86865234375, 0.001575469970703125, -0.39453125, 0.916015625, 0.05487060546875, -1.4482421875, 1.21484375, -0.229736328125, 0.9384765625, -0.378662109375, 0.22021484375, 0.242431640625, -0.59716796875, 0.446044921875, 0.0286865234375, 1.0361328125, 1.669921875, 1.779296875, 0.64599609375, -0.642578125, 1.2041015625, -1.0859375, -2.283203125, 1.298828125, 1.0830078125, 0.548828125, -0.78173828125, -0.92529296875, -1.5966796875, 1.076171875, 0.373046875, -1.4873046875, 0.257080078125, -1.8779296875, -0.87744140625, 0.93212890625, 1.9423828125, -0.9697265625, -0.7841796875, -1.025390625, 1.2119140625, -1.4521484375, -1.068359375, -0.3251953125, -0.91845703125, -0.017730712890625, 0.065673828125, 0.8583984375, 0.01300811767578125, -0.71826171875, -1.140625, 0.5986328125, 0.6416015625, 0.7255859375, -0.311279296875, -0.79345703125, 1.47265625, 0.86376953125, -0.53369140625, -0.477294921875, 0.02618408203125, 1.328125, 0.12115478515625, -0.7607421875, -0.047088623046875, -0.8916015625, 0.5400390625, -0.873046875, -0.66064453125, 0.4404296875, -1.0888671875, -0.80859375, 2.3125, -0.6083984375, -1.5185546875, 0.2314453125, -1.2783203125, -0.1640625, 0.061370849609375, -0.77734375, -0.288330078125, -0.78955078125, -1.1455078125, 1.224609375, 0.320556640625, 1.8583984375, -0.51220703125, 1.3037109375, -0.09320068359375, -0.8134765625, 0.68115234375, -0.12890625, 2.375, 1.7861328125, -0.8671875, 0.21630859375, -0.79052734375, -0.072265625, 1.138671875, -0.82177734375, 0.173828125, 0.472900390625, 0.978515625, -1.421875, 1.21875, 0.445556640625, -1.12109375, 1.171875, -0.4921875, 0.3037109375, 0.49560546875, 1.798828125, 1.015625, -0.15576171875, 0.6494140625, -0.045623779296875, -0.2159423828125, -2.263671875, 0.51171875, 0.42236328125, -0.342041015625, -0.2127685546875, -0.36669921875, -0.430908203125, 0.62548828125, -0.212890625, 0.154052734375, 0.12060546875, -0.12335205078125, -1.2373046875, -0.12255859375, -1.455078125, -2.52734375, 0.109375, 1.26953125, -0.14599609375, -0.5517578125, 0.103271484375, -0.0675048828125, 1.330078125, -1.9384765625, -2.025390625, -0.55224609375, 1.037109375, 0.62158203125, 0.2259521484375, -0.810546875, 1.5283203125, 0.11871337890625, -0.377197265625, -0.4814453125, 1.53515625, 0.87060546875, -0.560546875, 1.0947265625, -1.1240234375, -1.595703125, 0.2322998046875, 0.390869140625, -0.83837890625, -1.849609375, -1.255859375, 2.80859375, 1.849609375, 0.337646484375, 0.85302734375, -0.81201171875, -0.2254638671875, 0.6552734375, -0.242919921875, -1.025390625, -0.23046875, -1.5390625, 0.8447265625, 0.66455078125, -1.5185546875, 0.0782470703125, 0.1868896484375, -0.384765625, -0.79736328125, -0.62060546875, 1.08203125, 0.5341796875, -1.298828125, -0.796875, -0.7568359375, 1.298828125, 0.8564453125, 0.25, -0.22314453125, -2.10546875, 0.046600341796875, 0.73388671875, -0.1907958984375, -0.1851806640625, 1.234375, 0.06756591796875, 1.6669921875, 1.318359375, 0.7333984375, 0.60986328125, 1.0830078125, -0.2330322265625, 1.0693359375, 0.77490234375, -0.1539306640625, 0.103515625, -0.387451171875, 1.7783203125, -1.23046875, -0.45166015625, -0.260986328125, 0.1561279296875, 0.62744140625, 0.498291015625, 0.26025390625, 0.6064453125, 2.095703125, 0.1639404296875, -1.15234375, 1.77734375, 0.54833984375, -1.05859375, -0.476318359375, -0.009918212890625, -0.495849609375, -0.064208984375, 0.84423828125, -0.1221923828125, 1.986328125, 0.339599609375, -0.467529296875, -1.0029296875, 0.3955078125, -0.0643310546875, -0.5537109375, -0.08331298828125, 0.1290283203125, -0.468505859375, 0.438232421875, 1.8544921875, -1.7529296875, -0.7861328125, -0.599609375, 0.52490234375, -0.32666015625, -0.1236572265625, -0.79052734375, 0.2978515625, 0.2015380859375, 0.705078125, -0.89111328125, -2.419921875, -0.2451171875, -0.53515625, 1.439453125, -1.3330078125, 1.3486328125, 0.90185546875, 0.0687255859375, 0.07330322265625, -1.1416015625, 0.320068359375, -0.7236328125, -2.27734375, 0.1614990234375, 0.73486328125, -0.220947265625, -0.74755859375, 1.6708984375, 2.06640625, 0.491455078125, 1.7919921875, -1.84765625, 0.39111328125, -0.2119140625, -0.9091796875, -0.69189453125, 0.215576171875, 1.92578125, 0.1307373046875, 0.158935546875, 0.96923828125, 1.162109375, -1.087890625, -0.61376953125, 0.76953125, 0.7900390625, -0.0196990966796875, 0.35302734375, 0.414306640625, 0.92578125, -0.91552734375, 0.048492431640625, -0.31298828125, -0.08349609375, 1.083984375, -0.0155487060546875, 0.50830078125, -0.587890625, -0.62744140625, 0.5087890625, 0.51708984375, -0.67529296875, -0.060943603515625, 0.2056884765625, 1.2138671875, -0.451904296875, 0.086669921875, 0.67529296875, 1.9072265625, -0.4794921875, -0.73974609375, -0.783203125, 0.2177734375, -3.03125, -2.1015625, 0.60009765625, -0.69091796875, 0.6337890625, -1.1796875, -1.568359375, -2.30078125, -0.445556640625, 0.94921875, 0.445556640625, 0.35009765625, -2.3125, 0.65087890625, 0.5146484375, 1.484375, -0.4033203125, -0.216796875, -0.8271484375, 0.65478515625, -1.2158203125, 1.8115234375, -1.0810546875, 0.94970703125, 0.5859375, -0.0161590576171875, -0.90576171875, 0.6103515625, -0.64501953125, -0.3271484375, -1.6376953125, -1.7138671875, 0.5556640625, -1.6787109375, -0.5078125, 0.8544921875, 0.417724609375, -0.1177978515625, -0.287841796875, -0.8701171875, 0.435546875, -0.2156982421875, -0.126708984375, -0.26220703125, -1.0234375, -0.41162109375, -0.9599609375, 0.87646484375, 0.375244140625, 1.12109375, -0.0263214111328125, 0.1793212890625, 0.1566162109375, 0.06256103515625, -1.23828125, 0.07098388671875, 0.11236572265625, 0.99560546875, -1.779296875, -0.33251953125, 0.74169921875, 0.381103515625, 1.2890625, 0.1580810546875, 0.470703125, -0.92822265625, -0.4365234375, -0.8173828125, 0.72119140625, 0.28955078125, -1.53515625, -2.361328125, -1.6015625, -2.59375, -1.5517578125, 1.6337890625, 0.375244140625, 1.1884765625, -1.1328125, 0.302978515625, -1.6025390625, 0.6416015625, -1.173828125, -1.5595703125, 1.7900390625, 0.822265625, 0.11016845703125, 0.740234375, 0.21142578125, 0.07659912109375, -1.2099609375, -1.5380859375, 1.294921875, 0.232177734375, 0.49072265625, -1.255859375, -2.21875, -1.103515625, 1.865234375, 0.179931640625, 1.4140625, 0.045684814453125, 0.416748046875, -0.09307861328125, -1.435546875, -1.2470703125, 0.533203125, 0.450927734375, -1.5087890625, 0.63037109375, -0.59619140625, -0.010711669921875, 0.92041015625, 0.46923828125, 1.7705078125, 0.75390625, -0.880859375, 0.73974609375, 0.73388671875, -0.52294921875, -0.3955078125, -0.0712890625, -1.611328125, 0.2486572265625, -2.103515625, 0.488037109375, -0.0159149169921875, -0.2054443359375, 1.7822265625, 1.1953125, 1.064453125, -0.5947265625, -1.6767578125, 1.0546875, -0.91357421875, 0.239990234375, 1.46875, 0.0906982421875, 2.482421875, -0.35791015625, 0.460205078125, -2.134765625, 1.8779296875, 0.330078125, -0.28369140625, 0.136474609375, 0.8642578125, 1.1142578125, 1.8662109375, -0.54296875, -0.12200927734375, -1.455078125, -1.0712890625, -2.19921875, -0.65576171875, -1.0126953125, -0.0859375, -0.60791015625, -0.97314453125, 0.2607421875, 0.0396728515625, -1.173828125, 1.0283203125, 0.98779296875, 0.25146484375, 0.220703125, -0.456787109375, -0.2301025390625, -0.64404296875, 1.34375, 1.66015625, 0.1959228515625, -0.58349609375, -0.0916748046875, 0.857421875, -0.9970703125, -0.95703125, 1.7587890625, -0.447509765625, -0.169677734375, -0.2783203125, -0.322998046875, -0.79150390625, -0.399169921875, 0.2200927734375, -1.021484375, 0.93994140625, 0.5615234375, -1.712890625, 1.173828125, 0.60302734375, -1.021484375, 0.294921875, 0.728515625, -0.47802734375, -0.544921875, 1.71484375, -0.408203125, -1.3037109375, -1.484375, 1.2041015625, 1.6435546875, -0.8046875, 0.88232421875, 0.70654296875, 0.8388671875, 0.93701171875, 0.51708984375, 0.9189453125, 0.0137786865234375, -0.3837890625, -1.28515625, 0.54443359375, -1.3173828125, 1.2099609375, -0.8212890625, -0.57177734375, 1.2216796875, 0.38525390625, -1.19140625, 0.303466796875, -0.2900390625, 2.38671875, 0.70068359375, -1.177734375, 0.350341796875, 0.4921875, -0.434814453125, 2.04296875, -1.96875, -1.201171875, -1.169921875, 0.60791015625, -1.662109375, 0.92041015625, -0.7216796875, -0.89599609375, -0.96142578125, 1.0107421875, -0.51220703125, 0.39306640625, -1.9521484375, -0.481689453125, 2.330078125, 0.1072998046875, -0.11810302734375, -0.451904296875, 0.42529296875, -0.3662109375, -0.99560546875, 0.01177978515625, 0.1617431640625, -2.294921875, -0.1785888671875, -1.021484375, 0.8681640625, -0.548828125, -0.5703125, 0.919921875, -0.5322265625, -0.1627197265625, 0.1453857421875, -0.91943359375, -0.93701171875, -0.05572509765625, -0.74658203125, -1.1240234375, -0.2445068359375, 1.052734375, 1.1845703125, -0.66552734375, -1.5693359375, 1.7548828125, -0.1827392578125, 1.2177734375, -0.393798828125, -1.04296875, 0.77880859375, -0.673828125, 0.49658203125, 0.2105712890625, 0.24755859375, -0.316162109375, -0.642578125, 0.47314453125, 0.74365234375, 1.6005859375, -0.728515625, -0.10986328125, 0.283447265625, -1.8779296875, -1.8564453125, -1.3857421875, 0.2705078125, 1.1220703125, 0.328125, -0.50244140625, 0.3896484375, -1.28125, 1.1142578125, 0.73291015625, 2.453125, 0.300537109375, 0.1954345703125, -0.09820556640625, -0.81982421875, 1.72265625, 1.0888671875, 0.43798828125, -1.369140625, -0.603515625, 0.187744140625, 0.61669921875, 0.257080078125, 1.080078125, -0.228515625, 1.857421875, -0.287841796875, 0.44873046875, -0.39111328125, -1.4833984375, -1.1953125, 0.59375, -0.353759765625, -0.826171875, 0.9296875, -1.951171875, -1.17578125, -0.74365234375, 0.0841064453125, -0.456787109375, 0.9755859375, -1.0771484375, 0.68310546875, 0.41259765625, 1.4013671875, -0.092529296875, -0.82958984375, 1.0419921875, -1.5546875, 0.87451171875, 0.23388671875, 1.0498046875, 0.2120361328125, -0.323974609375, 0.89306640625, -1.5615234375, 1.140625, 1.5068359375, 0.59765625, -0.31591796875, 1.0810546875, -0.87255859375, 0.06103515625, 0.410888671875, 0.52001953125, -0.70703125, -0.340087890625, -1.3857421875, 1.16015625, 0.0838623046875, -0.331787109375, -0.0596923828125, -0.74658203125, -0.376708984375, 0.392822265625, -0.54541015625, 1.71484375, -0.951171875, 0.91845703125, 0.1546630859375, 0.317626953125, -0.0107421875, 0.82763671875, 0.92041015625, -0.480224609375, -0.01233673095703125, -0.94970703125, 0.30419921875, -0.222412109375, 0.7568359375, 2.693359375, -0.7109375, -0.53564453125, -1.1083984375, -1.734375, -0.03662109375, -1.2783203125, 1.31640625, 0.11029052734375, 1.4921875, 1.7587890625, -0.06195068359375, -0.60009765625, -1.0732421875, -0.0736083984375, 0.57958984375, -0.2802734375, -0.51025390625, 0.71826171875, 0.311279296875, -0.93115234375, -0.63037109375, 0.30419921875, 0.71240234375, 1.2119140625, 0.5087890625, -0.250732421875, -1.0654296875, -1.43359375, 0.280517578125, -0.423828125, 1.154296875, 1.052734375, -0.55078125, 0.227294921875, -1.3837890625, -1.0458984375, -0.271728515625, 0.037567138671875, -1.025390625, 2.3203125, 0.6044921875, 0.01534271240234375, -1.1220703125, 1.263671875, -0.448974609375, 0.437744140625, 1.0703125, -0.845703125, -0.043060302734375, -1.4521484375, -3.12109375, 2.029296875, -0.55908203125, 0.328369140625, -0.05950927734375, 0.255126953125, -0.88232421875, 0.505859375, -1.7607421875, -0.8408203125, -1.0546875, 0.45556640625, 0.09405517578125, 0.38671875, 1.173828125, -0.7236328125, 1.0615234375, -1.4482421875, 0.029876708984375, -1.4228515625, 0.329833984375, -0.75927734375, -2.46484375, 2.4375, -0.69775390625, 0.66259765625, 0.464599609375, 0.4462890625, -0.308837890625, 0.533203125, 0.379638671875, 1.0048828125, 0.16943359375, 0.80517578125, -0.312744140625, -0.642578125, -0.00035643577575683594, -2.205078125, -0.06964111328125, -1.146484375, -0.57568359375, 0.130859375, -1.201171875, 0.322509765625, 2.037109375, -1.07421875, -0.65234375, -0.3447265625, 0.431640625, -0.02294921875, 1.0380859375, -1.0380859375, 1.2333984375, 1.7412109375, -1.2509765625, -0.132568359375, -0.435546875, 2.333984375, 0.1964111328125, 0.041961669921875, 1.0419921875, 1.3154296875, -0.2281494140625, -0.27880859375, -0.87548828125, 0.982421875, -0.68896484375, 1.296875, 0.70068359375, -0.06591796875, 0.1129150390625, 0.92529296875, 1.3828125, 0.04638671875, -0.7529296875, -0.282958984375, 0.4140625, 0.74609375, 0.259521484375, 0.5634765625, 2.4453125, -0.305908203125, -0.83837890625, -0.0357666015625, 0.8740234375, -0.156005859375, -0.66357421875, 0.88818359375, 0.828125, 1.1494140625, 2.27734375, 1.5810546875, -0.91552734375, -0.9013671875, -0.292724609375, 0.46875, -0.31982421875, -0.4248046875, 1.0029296875, 1.71875, -0.8955078125, -1.2890625, -0.019287109375, 0.12225341796875, -0.69677734375, -1.5888671875, 0.340087890625, 0.3447265625, -0.79248046875, 1.5966796875, -0.435546875, 0.6044921875, -1.119140625, -0.226318359375, 0.666015625, 1.5908203125, -0.08917236328125, -0.99169921875, -0.315185546875, -1.2099609375, -0.9267578125, -0.20263671875, 1.423828125, -1.2939453125, 1.1025390625, 0.3740234375, -0.2529296875, -0.5009765625, 1.919921875, -1.27734375, 0.70166015625, -0.0111083984375, -0.689453125, 0.5517578125, 0.2279052734375, 1.8271484375, 0.6630859375, 1.083984375, 0.315673828125, -0.46728515625, 0.059539794921875, -1.1220703125, 0.1468505859375, 0.457763671875, 0.1961669921875, -0.677734375, 0.970703125, -0.10986328125, -1.61328125, 0.402587890625, 1.6044921875, -0.24609375, 0.7119140625, -1.1005859375, -0.93408203125, -0.42333984375, 0.441162109375, -1.0009765625, -0.9697265625, 0.1796875, 1.9541015625, -0.64599609375, 0.09674072265625, 0.28466796875, -0.44287109375, -0.88623046875, -0.27001953125, 0.884765625, -0.66015625, -0.316162109375, 0.83203125, -1.068359375, 1.2861328125, -0.2373046875, -0.67626953125, 0.72705078125, 1.51953125, 0.2239990234375, 1.154296875, 0.19140625, 0.34375, 0.67138671875, 2.083984375, -2.00390625, 0.60205078125, 0.1300048828125, -1.748046875, -0.75341796875, -0.2022705078125, 0.3642578125, -1.53125, -0.2880859375, -0.625, -0.367431640625, 0.05712890625, 0.7158203125, 0.357421875, -1.3955078125, -0.9990234375, -0.8369140625, 0.309814453125, -0.0872802734375, -1.263671875, 0.53173828125, 0.73828125, -0.0382080078125, -0.8759765625, 0.0259552001953125, -3.09765625, 1.275390625, -1.33984375, 1.70703125, 0.7626953125, -0.8876953125, -0.7900390625, -2.6328125, -0.413818359375, -0.68896484375, -0.5693359375, 1.23046875, 0.1689453125, 0.16162109375, -0.296630859375, -0.580078125, -0.96630859375, -0.75, 1.181640625, -1.23828125, -0.64501953125, -0.5478515625, 0.210205078125, -0.34619140625, 1.6826171875, -0.5810546875, -0.41357421875, 0.12371826171875, 1.4482421875, -0.77880859375, 0.6708984375, 0.0330810546875, 2.462890625, 1.4208984375, 0.0112762451171875, -0.60693359375, 1.77734375, 0.64697265625, 0.41845703125, 0.1378173828125, 1.7705078125, -0.78857421875, 2.55859375, -0.4814453125, 0.279541015625, -0.99365234375, -0.79541015625, 1.708984375, -0.73486328125, 0.5478515625, 1.2880859375, 0.117919921875, -1.4521484375, -1.63671875, -0.751953125, 0.0592041015625, -0.5068359375, -1.7978515625, 1.162109375, -0.85888671875, 1.220703125, -0.92431640625, 0.2109375, -0.1500244140625, -1.4931640625, 0.283447265625, -0.39306640625, 1.5263671875, -0.42822265625, 0.64599609375, -1.4453125, 1.21484375, -1.326171875, 2.05078125, 1.140625, -1.86328125, -0.5556640625, -2.0078125, -0.6396484375, 0.8251953125, -0.1807861328125, 2.05078125, 1.2099609375, 0.1456298828125, -0.489501953125, 1.2666015625, -0.26025390625, -1.3310546875, -0.96044921875, 0.04150390625, 0.9892578125, 1.49609375, 1.6494140625, 0.7578125, 0.4345703125, 1.9677734375, 1.8984375, -1.2236328125, -0.8271484375, 0.2484130859375, -0.90966796875, 1.857421875, 0.6787109375, -0.88525390625, 0.81494140625, 0.62060546875, 0.03924560546875, 0.66943359375, 0.114990234375, 2.556640625, 0.3935546875, 1.21484375, -0.054412841796875, 0.2227783203125, 0.02325439453125, 0.48681640625, 0.443603515625, -1.732421875, -0.200439453125, 0.4169921875, 0.5263671875, 0.0794677734375, -0.17041015625, 0.0034236907958984375, -0.84130859375, 0.168212890625, 1.4091796875, -0.564453125, 0.51953125, 2.3203125, -1.4423828125, 1.4453125, -0.5615234375, 1.13671875, 0.1529541015625, 0.5107421875, 1.513671875, -0.308349609375, -0.70751953125, -0.241943359375, -0.0838623046875, 1.2001953125, 1.0791015625, 0.37158203125, -0.830078125, -1.5048828125, 0.049407958984375, 0.136474609375, 2.076171875, 1.5498046875, -2.609375, 0.25390625, 1.88671875, -1.3798828125, 0.6201171875, -0.322265625, -0.81005859375, 0.267333984375, -1.7890625, 1.26953125, -0.5302734375, 0.70556640625, 0.2091064453125, 0.517578125, -1.033203125, 0.65673828125, -2.1953125, 1.0712890625, 0.2200927734375, -0.66064453125, -0.501953125, 0.8046875, 0.91455078125, 1.53515625, -1.046875, 0.83544921875, 0.492431640625, -0.541015625, -0.97412109375, -0.31298828125, -1.4208984375, 0.41650390625, -0.08148193359375, 0.0205078125, -0.55712890625, 0.1676025390625, 1.455078125, 0.374755859375, 1.6337890625, -0.464111328125, 0.4287109375, -2.3203125, -0.119140625, 1.4306640625, 1.7724609375, -0.921875, -0.701171875, -1.5537109375, 2.05859375, 0.9140625, -0.389404296875, 1.1533203125, 1.0947265625, -0.73779296875, -0.252197265625, -1.4970703125, 0.4765625, 0.92919921875, -1.376953125, -0.5166015625, 1.1982421875, -0.1221923828125, 1.9619140625, -0.17626953125, -0.73095703125, -0.365234375, 0.7958984375, 1.4228515625, -0.9169921875, -0.1492919921875, -0.83447265625, -1.431640625, 0.03436279296875, -1.6875, -2.134765625, 0.71435546875, 0.767578125, 1.1982421875, 0.3115234375, 0.202880859375, 0.8466796875, 0.5615234375, 0.6103515625, 1.16015625, 1.2138671875, 0.5927734375, -1.4306640625, 1.361328125, -0.1495361328125, -0.22998046875, -0.014801025390625, -0.457763671875, -0.1632080078125, -0.3916015625, 0.2171630859375, 0.65234375, 1.099609375, -1.3857421875, -0.6728515625, -0.311279296875, -0.7099609375, -1.681640625, -0.818359375, 1.37109375, -0.6796875, 0.1798095703125, 1.0087890625, -0.53857421875, -0.63525390625, -0.355712890625, -0.1923828125, -0.66552734375, -2.447265625, 0.1904296875, 1.087890625, 0.423095703125, 0.88037109375, 0.322021484375, 0.25341796875, 1.2802734375, -2.365234375, -2.12109375, -0.57470703125, 0.841796875, -0.63916015625, -0.1380615234375, -1.7724609375, 0.5478515625, -2.201171875, 0.455810546875, -1.4140625, 1.0107421875, 0.83935546875, -1.451171875, 1.607421875, -1.3076171875, 1.2998046875, 2.201171875, 1.359375, -1.3046875, -0.90966796875, 0.239501953125, -0.5390625, -0.41015625, -1.0224609375, -0.98779296875, 0.34619140625, 0.096923828125, -0.3466796875, -0.59814453125, 0.31103515625, 0.00467681884765625, 0.1900634765625, 0.056884765625, -0.6376953125, 0.33837890625, -1.0771484375, -0.319091796875, 0.544921875, -0.62890625, -0.744140625, -0.23876953125, 0.9873046875, 0.18310546875, 0.98779296875, 0.33349609375, -0.56298828125, -0.62939453125, -0.118896484375, -0.61376953125, -0.135986328125, -0.2193603515625, 0.58349609375, 0.76611328125, 2.412109375, -0.92529296875, 0.07196044921875, -0.1728515625, 0.43603515625, -0.56787109375, -0.9306640625, 0.420654296875, 0.9599609375, -0.2220458984375, -1.2138671875, 0.02532958984375, 0.7900390625, -0.048431396484375, 1.2880859375, 0.61181640625, -0.346435546875, 0.197509765625, -0.7509765625, 0.030120849609375, 0.52197265625, -2.033203125, 0.94873046875, 0.7314453125, -0.92529296875, -0.703125, 2.12109375, 1.029296875, 0.8095703125, -0.292236328125, -0.1058349609375, -0.0877685546875, -1.1904296875, -1.970703125, 0.6396484375, -1.0341796875, -0.316650390625, 1.525390625, 0.030242919921875, -0.96826171875, 0.5126953125, 1.1728515625, -1.173828125, 0.708984375, 0.51171875, -0.415283203125, 0.469482421875, 0.94775390625, 0.2413330078125, 0.9140625, 1.7314453125, 0.97705078125, -0.125244140625, 0.07952880859375, 0.70654296875, -0.26171875, -0.63427734375, 0.016387939453125, -0.00836181640625, 0.79736328125, -0.40966796875, -0.71923828125, 0.67626953125, 1.22265625, 0.278564453125, -0.578125, 0.368896484375, -0.06561279296875, 0.464111328125, 1.072265625, 1.5927734375, -0.1651611328125, 2.1171875, -1.04296875, -1.109375, -1.0830078125, 0.626953125, -0.97900390625, 0.853515625, -1.9306640625, -0.75830078125, 0.451171875, 0.599609375, -0.80419921875, 1.306640625, -0.33837890625, 1.140625, -0.401611328125, 0.85888671875, 0.0701904296875, 0.7763671875, -0.14794921875, -0.01044464111328125, 0.449951171875, -0.74072265625, -1.0556640625, 0.262939453125, 1.68359375, 0.08685302734375, 0.466064453125, 0.77099609375, -1.4287109375, -0.80712890625, 2.255859375, -1.64453125, 2.5078125, -1.80078125, 0.41064453125, -0.409423828125, -0.320556640625, 1.103515625, -0.049652099609375, 2.291015625, 0.73095703125, -2.072265625, -0.59423828125, 0.7822265625, 0.97998046875, -0.046783447265625, 0.29345703125, -0.8271484375, 1.4130859375, 0.070556640625, 0.6650390625, -0.546875, -0.03179931640625, -0.34521484375, 0.40576171875, -0.08209228515625, -1.4189453125, -0.466552734375, -0.252685546875, -0.8857421875, -0.798828125, -0.296875, -2.3515625, -0.401611328125, 0.68408203125, -0.1888427734375, 0.158935546875, -0.5234375, -0.77587890625, -0.076171875, -1.8798828125, 1.060546875, 0.97900390625, 1.8701171875, 0.375, 2.935546875, -0.83251953125, 1.9794921875, 0.71484375, 0.0712890625, -0.432373046875, -0.25341796875, -0.137451171875, -0.28759765625, -0.69873046875, 2.076171875, 0.6796875, -0.230224609375, 0.98583984375, -0.71240234375, 1.7470703125, -0.06866455078125, 1.1025390625, -2.45703125, -1.1884765625, -1.462890625, -0.49072265625, -1.18359375, -0.5185546875, -0.95751953125, 2.203125, -0.07550048828125, -1.1083984375, -1.6962890625, 0.30126953125, -0.4794921875, 0.9326171875, -0.453125, 0.265380859375, 0.005702972412109375, -1.5888671875, 0.69873046875, -1.4541015625, 1.3740234375, -0.9716796875, 0.85498046875, -0.2061767578125, -0.2056884765625, -1.30078125, 0.658203125, 1.2275390625, -1.2080078125, -0.020904541015625, 0.74365234375, -0.155029296875, -1.470703125, -0.91748046875, -0.0091400146484375, -1.9228515625, -1.388671875, -1.2314453125, 1.974609375, -0.08966064453125, -0.3359375, 1.9423828125, -0.5234375, 0.478271484375, 0.740234375, 2.072265625, 0.280029296875, -1.1572265625, -0.83935546875, 1.5673828125, 0.160888671875, -0.1829833984375, -0.1767578125, 0.5361328125, -0.410888671875, -1.2841796875, -0.217041015625, 0.921875, -0.43310546875, 0.41455078125, 0.74658203125, -0.572265625, 1.037109375, 0.487060546875, -0.72900390625, -0.1990966796875, 0.66748046875, 0.0034122467041015625, 0.6962890625, -1.2109375, -0.5517578125, 0.0027523040771484375, -1.0517578125, 0.94482421875, 0.2186279296875, -1.8857421875, -0.7275390625, 0.88330078125, 1.048828125, 2.01171875, -0.47705078125, 0.72998046875, -0.364990234375, 0.0699462890625, 0.2666015625, -1.2255859375, -0.03350830078125, -1.056640625, -0.53076171875, -0.67138671875, -1.5029296875, 0.625, 0.9208984375, 0.300537109375, 0.63330078125, -0.2337646484375, -1.2568359375, -0.10302734375, 0.36572265625, -2.021484375, 0.88525390625, 0.448486328125, -2.4140625, -0.3486328125, -1.3369140625, -0.28955078125, -0.2890625, 0.28564453125, -0.52197265625, 0.08575439453125, 0.291259765625, 0.79150390625, -1.115234375, 0.2161865234375, 0.006572723388671875, -1.9140625, -0.072509765625, 0.491455078125, -0.12420654296875, 1.154296875, 1.7509765625, 1.4345703125, -0.313720703125, -0.8154296875, 0.1502685546875, -1.2578125, -1.7470703125, 0.609375, 0.8740234375, -0.07000732421875, 0.931640625, 1.650390625, 0.5537109375, -0.96337890625, -0.36865234375, -0.54296875, 0.247802734375, 0.2105712890625, -0.8212890625, 0.76318359375, -2.3828125, -0.57470703125, 0.1064453125, -1.83984375, -0.5927734375, 0.1719970703125, -1.0087890625, 0.083740234375, -0.490966796875, 0.290771484375, -0.1717529296875, 0.27099609375, 0.08917236328125, 0.76220703125, 0.12286376953125, -0.9140625, 1.2666015625, -1.0283203125, -0.55712890625, -0.921875, 0.8056640625, -0.01013946533203125, 0.166015625, 0.92724609375, -0.53564453125, 0.29052734375, -0.0738525390625, -1.2041015625, 1.376953125, -0.578125, 1.36328125, 1.2568359375, -0.206787109375, 0.60888671875, -0.390380859375, -0.84765625, 1.47265625, 0.38818359375, -0.1865234375, 0.9453125, 0.61962890625, -2.0859375, -1.2939453125, -0.493896484375, 1.9072265625, 0.1551513671875, 0.93359375, 1.498046875, 0.56494140625, -0.11724853515625, -0.916015625, 1.013671875, -0.368896484375, 0.04693603515625, -0.11651611328125, 0.341064453125, 0.371826171875, -0.1319580078125, -1.5576171875, 2.5, 0.6025390625, -0.329345703125, -0.75, 0.2451171875, 0.0228729248046875, -0.5869140625, 0.296875, -0.38525390625, -2.326171875, -0.08050537109375, -0.7431640625, 0.88525390625, 0.1917724609375, 1.3271484375, 0.703125, -0.58544921875, 0.339599609375, 1.12890625, -1.0361328125, 2.310546875, 0.79541015625, -0.1514892578125, 0.50390625, 0.267578125, -0.7822265625, 0.185546875, 1.8994140625, 0.70751953125, -0.263916015625, -0.80810546875, 0.8046875, -0.98583984375, 0.26220703125, 0.171875, 0.418701171875, -1.3505859375, -0.347412109375, 1.2705078125, 0.07501220703125, 0.60107421875, 0.74951171875, 1.615234375, 0.13134765625, -0.65771484375, -0.339111328125, -0.740234375, 0.40869140625, 1.4775390625, -0.449462890625, -0.96826171875, 1.8115234375, -0.1279296875, -0.77490234375, 1.7275390625, 0.267822265625, -0.133056640625, -0.82763671875, 0.99609375, 2.07421875, -0.1441650390625, 0.361328125, -0.67626953125, 1.0009765625, 0.09844970703125, 1.0361328125, 0.040283203125, -0.332275390625, 0.87158203125, -0.82080078125, 0.802734375, -1.5234375, -0.5107421875, 1.185546875, 0.0848388671875, 0.734375, 0.316650390625, -0.1817626953125, 1.095703125, -1.21875, 0.269287109375, 0.1097412109375, 0.1646728515625, -1.318359375, -2.224609375, -0.53564453125, 0.28955078125, -0.41650390625, 0.112548828125, -0.97216796875, -1.3935546875, -0.11260986328125, 0.06878662109375, 0.1270751953125, -0.0269012451171875, 0.175537109375, -1.919921875, -0.49462890625, -0.452880859375, 1.251953125, -2.693359375, 0.68505859375, 1.298828125, 1.4697265625, -0.80322265625, -1.1982421875, 0.70166015625, -1.326171875, 0.255615234375, -1.69140625, 1.6533203125, -0.29443359375, -0.5, 0.96337890625, -0.86083984375, -2.9375, -0.541015625, -1.232421875, -0.83935546875, -0.20751953125, -0.55908203125, -0.78271484375, -1.8505859375, -0.6015625, -0.020904541015625, -2.580078125, -0.361328125, -0.369873046875, 0.61376953125, 0.7314453125, 0.373291015625, -1.04296875, -2.359375, -0.14794921875, -0.345947265625, 1.0322265625, -0.25634765625, 1.392578125, -1.1396484375, 0.1798095703125, -0.412841796875, -0.66357421875, 0.8857421875, 1.03125, -1.0517578125, -0.998046875, 0.80126953125, -1.724609375, -0.1162109375, -0.7109375, -0.2178955078125, -0.5654296875, 1.173828125, -1.1376953125, 1.671875, 0.406494140625, 0.27490234375, 0.5185546875, -1.439453125, 1.1494140625, 0.28515625, 1.1845703125, 0.76416015625, -0.335693359375, 0.59228515625, -0.185302734375, -0.6220703125, -0.626953125, 0.329345703125, -1.982421875, -1.8740234375, -0.57470703125, 0.1666259765625, -1.041015625, -2.083984375, 0.86376953125, 0.0665283203125, 0.9775390625, 0.85986328125, 0.5947265625, -1.82421875, -0.290771484375, -0.88134765625, -1.1357421875, 0.38671875, 0.45849609375, 0.02294921875, 1.794921875, 0.66845703125, -1.1083984375, -0.86669921875, -2.697265625, -0.298095703125, 1.5673828125, 0.20263671875, -0.154541015625, 1.23046875, -1.4765625, -1.5478515625, -0.1632080078125, -1.16015625, 0.1968994140625, 0.7734375, 1.8046875, -1.1953125, 1.0146484375, 0.4580078125, -0.299072265625, 0.8037109375, 1.470703125, -0.78271484375, -0.24560546875, -0.4306640625, 0.97607421875, -0.1009521484375, -0.178955078125, 0.497314453125, -0.437744140625, 0.58837890625, -0.77490234375, 0.1768798828125, 1.552734375, 1.044921875, -1.5830078125, 0.0650634765625, 0.43310546875, 0.08740234375, -0.26123046875, -0.72216796875, -0.36767578125, 0.71337890625, -1.615234375, 0.1744384765625, -0.004413604736328125, 0.97021484375, 1.2861328125, -1.12109375, 0.354248046875, 0.962890625, 0.6337890625, -0.5810546875, -1.271484375, 1.7099609375, 0.1820068359375, 0.349365234375, 0.65283203125, -0.338623046875, -0.9296875, -0.83642578125, 0.095947265625, 1.2734375, -0.75927734375, 1.265625, 1.7041015625, -1.384765625, 0.3896484375, -0.50439453125, 0.060577392578125, -1.0791015625, -0.3505859375, 0.77978515625, 0.470458984375, 0.250244140625, 0.84326171875, 0.075439453125, -0.391357421875, 0.221435546875, -0.67431640625, -1.1611328125, 0.291748046875, -3.060546875, 1.9013671875, -0.603515625, -0.13671875, -0.87646484375, 0.1431884765625, 0.68994140625, 0.47119140625, -1.271484375, 0.43359375, -0.912109375, 0.369140625, -2.25, 0.51953125, 0.02301025390625, -0.76416015625, -1.0849609375, 0.281005859375, -0.7822265625, 1.1337890625, -0.97216796875, -0.8056640625, -0.0011844635009765625, 0.716796875, -0.8095703125, 0.94970703125, -0.35693359375, -2.125, -0.91259765625, -0.9326171875, 2.45703125, 0.036163330078125, -1.181640625, -1.5625, -1.04296875, 0.95849609375, 0.093994140625, 0.06256103515625, 1.2216796875, 0.30517578125, 0.83935546875, 1.1162109375, 0.794921875, 0.0191497802734375, 0.56005859375, 0.46435546875, -0.9677734375, 0.90966796875, -1.5908203125, 1.5224609375, 0.57763671875, -1.05078125, 0.2685546875, -0.92041015625, -0.46630859375, -0.2423095703125, -0.4140625, -1.0546875, 1.408203125, -0.681640625, -0.50537109375, 0.68603515625, 0.1361083984375, 0.09027099609375, 0.775390625, -1.029296875, -0.1710205078125, -1.12890625, -0.310546875, 1.6015625, 0.77392578125, 0.251220703125, 0.57470703125, -1.701171875, 1.92578125, 0.173583984375, 0.1256103515625, -1.083984375, -0.2139892578125, 0.7568359375, -0.896484375, 1.255859375, -0.27197265625, 1.130859375, -1.166015625, 0.3232421875, -0.10723876953125, 0.083984375, 0.69287109375, -1.07421875, -1.453125, -1.005859375, 0.6025390625, 0.396484375, -0.359619140625, -1.29296875, -0.7265625, -0.12103271484375, -0.0452880859375, -0.4140625, -0.81201171875, 0.2113037109375, 0.5224609375, -0.315673828125, -0.1912841796875, -1.0986328125, -0.92724609375, -1.0390625, 0.5107421875, -2.25390625, 0.896484375, 1.3779296875, 0.373046875, 1.11328125, -0.8466796875, -1.37890625, 0.1463623046875, 0.0435791015625, -2.01171875, -0.293212890625, 1.2548828125, -0.90966796875, -0.078125, -0.61376953125, 0.3095703125, 1.416015625, -0.58056640625, 0.00542449951171875, 0.6611328125, -1.79296875, -0.1773681640625, 0.259765625, -0.9716796875, -0.4072265625, -0.484619140625, 0.30224609375, 0.03167724609375, 0.33056640625, -0.171630859375, -1.01953125, -0.295654296875, -0.398193359375, -0.6748046875, -0.9453125, 1.8642578125, 0.434814453125, -1.4990234375, -0.505859375, 0.2493896484375, -0.46533203125, 0.227294921875, -0.9638671875, 0.45458984375, 0.05767822265625, -1.5361328125, 1.384765625, -0.0936279296875, 0.6630859375, 0.796875, 1.728515625, -1.0595703125, 0.11187744140625, 1.138671875, -0.77734375, -1.3505859375, 0.429443359375, 0.6298828125, -0.08514404296875, 0.5029296875, -0.1591796875, 0.390869140625, -1.3486328125, -0.1171875, -0.0872802734375, 0.4072265625, -0.2783203125, 0.63037109375, -0.380126953125, 0.73095703125, -0.61083984375, 0.4462890625, -0.263427734375, -0.580078125, -1.1474609375, 0.12091064453125, 0.482666015625, -1.17578125, -1.86328125, 2.109375, 0.51318359375, -0.58154296875, -0.0347900390625, -1.7880859375, -0.2020263671875, 1.1328125, -0.2666015625, -0.325927734375, 0.370361328125, -1.076171875, 0.92236328125, 0.464599609375, -0.6171875, 1.22265625, 0.06787109375, 0.180419921875, 2.029296875, 0.0289764404296875, -0.322265625, 0.84521484375, -0.367919921875, 0.33447265625, 0.90234375, -0.67138671875, -1.33203125, -0.546875, 0.386962890625, 0.442138671875, -0.276611328125, 0.3662109375, 0.994140625, 1.0556640625, 1.400390625, -1.435546875, 0.47216796875, -0.57763671875, -2.099609375, -0.9375, 0.59228515625, 0.3359375, 1.51953125, 0.74658203125, -0.978515625, 2.498046875, 2.390625, -1.119140625, 0.153564453125, 0.216552734375, -0.076416015625, 0.058349609375, -0.002288818359375, -0.255126953125, -0.001007080078125, -1.0068359375, 1.45703125, 0.501953125, -0.46923828125, 1.775390625, -0.357177734375, -0.426513671875, -1.23828125, 0.316162109375, 0.54833984375, 0.1844482421875, -0.336669921875, 2.060546875, -1.0322265625, -0.3125, -0.4140625, -0.61181640625, 1.27734375, 0.02923583984375, -0.5458984375, -0.2467041015625, 0.03570556640625, 0.1236572265625, 1.3212890625, 1.5341796875, -0.128173828125, -1.7451171875, 1.3193359375, 1.0751953125, -1.353515625, -1.1396484375, -0.0127410888671875, -0.359130859375, 0.57177734375, 3.49609375, -0.5009765625, -0.75830078125, 0.365966796875, -1.8427734375, -0.0799560546875, 0.032958984375, -0.79345703125, -0.271240234375, -2.283203125, 1.201171875, -1.8515625, -0.27880859375, -3.2109375, -0.869140625, -0.560546875, 0.97412109375, -0.302490234375, 0.79443359375, 0.7646484375, -1.310546875, 2.921875, -0.005031585693359375, -0.488037109375, 1.005859375, -0.1861572265625, -0.8017578125, -0.1634521484375, -0.78955078125, -0.97705078125, 1.140625, -0.79296875, 2.373046875, -1.529296875, 0.73876953125, -0.1593017578125, -0.84619140625, 0.84228515625, -1.17578125, -0.312744140625, -0.51171875, -0.7431640625, -0.90380859375, 0.26025390625, 0.73876953125, 0.2227783203125, -0.96875, 0.458740234375, 0.08331298828125, -1.2685546875, 0.041229248046875, -2.640625, 0.5302734375, 0.036651611328125, 0.767578125, 0.0025005340576171875, 1.919921875, -1.609375, 0.66796875, 1.861328125, -1.330078125, 0.84521484375, 0.98046875, -0.60400390625, -0.6650390625, -1.2783203125, 0.058135986328125, 0.1405029296875, 0.31494140625, -0.034820556640625, -1.09765625, 0.72021484375, 0.7548828125, 0.95458984375, -0.1451416015625, 0.69775390625, -0.28466796875, -1.0703125, -0.6220703125, -0.8701171875, 0.62060546875, 0.966796875, -1.4658203125, -1.904296875, -0.83642578125, 0.7431640625, 0.037384033203125, -0.8642578125, 2.30859375, -0.62646484375, -0.640625, -0.211669921875, 0.81689453125, 1.4580078125, 0.1202392578125, -0.66748046875, 0.3349609375, -1.10546875, -1.2607421875, 1.060546875, -0.5224609375, 2.703125, -1.4951171875, 0.45166015625, -0.94970703125, 0.264892578125, 2.20703125, -1.0703125, -0.98095703125, 0.1151123046875, 1.1552734375, -0.947265625, -0.262451171875, -0.14404296875, 1.06640625, -0.95068359375, -0.427490234375, 0.2459716796875, 1.42578125, -0.0306549072265625, 0.71240234375, 1.67578125, 0.5517578125, 0.70361328125, 0.319580078125, 0.044921875, -1.0751953125, 0.076416015625, -0.595703125, -1.0791015625, 0.8779296875, 0.1119384765625, 0.288330078125, -0.49169921875, 1.48046875, 1.1728515625, 0.16162109375, 0.1470947265625, 0.2362060546875, -0.7119140625, -1.3544921875, -0.390380859375, 1.162109375, 0.25048828125, 0.89794921875, -0.71728515625, 1.9931640625, -0.61474609375, -0.490234375, -1.8671875, -0.73095703125, 1.0185546875, -1.244140625, 1.2099609375, 0.513671875, -0.6162109375, 0.1995849609375, -0.462646484375, 0.71142578125, 1.1796875, -0.6337890625, 0.7841796875, -0.548828125, 0.9404296875, -0.5048828125, -1.6318359375, -3.26171875, 1.2470703125, -0.311279296875, 0.21435546875, -0.2059326171875, -3.357421875, -1.9970703125, 0.9228515625, 0.057373046875, -1.31640625, -1.396484375, 0.6552734375, -0.83251953125, -0.591796875, 0.75830078125, 0.7763671875, -1.443359375, 0.7490234375, -0.325439453125, 1.02734375, -0.146484375, -0.128662109375, 0.0095367431640625, 0.39599609375, 0.1873779296875, 2.26953125, 1.021484375, 0.101318359375, 0.6220703125, 1.92578125, 1.08984375, 1.3623046875, -0.08251953125, 0.0574951171875, -0.245849609375, -0.8828125, -1.673828125, 0.2012939453125, 0.352783203125, -0.84521484375, -1.41015625, -1.5302734375, 2.548828125, 0.97705078125, -0.37548828125, -1.05078125, 1.08984375, -0.90478515625, 1.0869140625, -0.4775390625, -1.4541015625, 0.22705078125, 1.8759765625, 1.380859375, -1.20703125, -1.07421875, -1.51953125, -2.314453125, 0.8388671875, -1.0732421875, 0.412841796875, -1.4384765625, 1.4638671875, -0.08416748046875, 0.2220458984375, 2.369140625, -0.287841796875, -1.888671875, -0.1343994140625, -0.4501953125, 1.138671875, -1.3154296875, 0.9560546875, 2.28515625, 1.6689453125, 0.2099609375, 0.472900390625, -0.031463623046875, 1.4853515625, -0.95458984375, 0.74169921875, 0.34619140625, 0.02764892578125, -2.041015625, -1.1220703125, 1.1943359375, 0.1461181640625, -0.1279296875, 0.05853271484375, 0.8369140625, -0.99560546875, -1.7197265625, -1.2470703125, 0.06671142578125, 0.178466796875, -0.190673828125, 0.99755859375, -0.74609375, -0.65869140625, -1.4560546875, 0.86767578125, 1.0009765625, 1.0859375, -0.46826171875, -0.10345458984375, 0.544921875, 0.09271240234375, -0.765625, -1.478515625, -0.4267578125, 0.0147857666015625, -1.8994140625, 0.84033203125, 0.48193359375, 0.2003173828125, -0.337890625, 0.52734375, 1.7978515625, 1.029296875, -1.4580078125, -1.41015625, -0.94775390625, 0.19921875, 1.515625, -0.78515625, -1.0625, -0.11083984375, 0.4921875, 1.5107421875, 0.205078125, -0.390380859375, -1.0205078125, -0.062042236328125, -1.83203125, 0.650390625, -0.7880859375, 1.2890625, 0.61474609375, 0.96142578125, -1.349609375, 0.64892578125, -0.311767578125, 0.351806640625, -0.444580078125, -2.1328125, -1.2763671875, 1.4609375, 0.491943359375, 0.380615234375, -0.66748046875, -1.3359375, 0.22265625, -0.048736572265625, 0.55810546875, -1.6650390625, -0.365234375, 0.888671875, 0.69873046875, -0.666015625, -0.2181396484375, -0.393798828125, -0.462646484375, -0.68310546875, 0.454833984375, -0.0474853515625, -0.2919921875, -0.5400390625, 1.8583984375, -2.162109375, -0.426513671875, 1.0888671875, 1.1357421875, -2.65625, -0.740234375, -0.509765625, -1.3583984375, 0.35400390625, 1.5703125, -1.1611328125, 1.0068359375, -1.4150390625, 0.90966796875, -0.89990234375, -1.66015625, -0.4775390625, -1.0029296875, 0.9140625, -0.89013671875, -0.75048828125, 0.2257080078125, 2.150390625, 2.431640625, 0.880859375, -0.181884765625, 0.90966796875, -0.1700439453125, -1.9248046875, -0.429931640625, 0.85302734375, -0.1512451171875, 0.34033203125, 0.763671875, -0.81640625, -1.130859375, 1.3017578125, 1.4638671875, -0.5048828125, -0.70458984375, -0.0648193359375, 1.1884765625, -0.76611328125, -1.517578125, -0.398681640625, 0.25390625, -0.11199951171875, -0.39404296875, -0.56298828125, 2.40625, -0.9755859375, -0.1787109375, -0.9453125, -0.69091796875, 0.88525390625, -1.0244140625, -0.748046875, -0.61181640625, 1.7001953125, 1.150390625, -1.046875, 0.15234375, -0.4609375, -0.3359375, 0.78955078125, -0.336181640625, 0.33154296875, -0.04931640625, -1.224609375, -0.02362060546875, -0.5556640625, 0.037750244140625, 0.8037109375, 0.68408203125, -0.6337890625, 0.178955078125, 0.59521484375, 2.1953125, -1.595703125, -0.0245208740234375, 1.1806640625, -0.86083984375, -0.6533203125, -0.919921875, 0.027191162109375, -0.4775390625, -0.79638671875, -1.1728515625, 0.8876953125, -0.69384765625, -0.8935546875, 0.0070037841796875, 0.09051513671875, 0.74853515625, -1.3466796875, -0.79541015625, -0.1275634765625, 0.05767822265625, 1.376953125, -0.12408447265625, 0.049468994140625, 1.5107421875, -0.8935546875, 0.440673828125, -0.289306640625, -1.259765625, 0.5908203125, 0.001903533935546875, 1.119140625, 1.8779296875, -0.2064208984375, 0.51953125, 0.7548828125, -0.3955078125, 0.98876953125, -0.07135009765625, -0.85107421875, -0.387451171875, -0.62939453125, -0.2364501953125, -0.455078125, -0.1925048828125, -0.9404296875, -2.181640625, -0.22509765625, 1.1982421875, -0.345947265625, 0.05328369140625, 0.37744140625, -0.94873046875, 1.5283203125, -0.45849609375, 0.58642578125, -0.058563232421875, -0.78662109375, -0.0355224609375, -1.1513671875, -1.349609375, 0.319580078125, 1.41015625, -1.1767578125, 0.09283447265625, 0.443359375, -0.154052734375, -0.11846923828125, -0.6396484375, 0.81787109375, 0.548828125, 2.0078125, 0.57763671875, -0.6005859375, -0.030303955078125, -0.1490478515625, 0.288818359375, 0.349853515625, 0.93701171875, -0.175537109375, -0.047576904296875, 0.8974609375, -1.7646484375, 0.472900390625, 0.61181640625, 0.230712890625, -0.72607421875, 0.1812744140625, -0.951171875, -0.66357421875, 0.300048828125, 1.5263671875, 0.239990234375, 0.8740234375, 0.92041015625, -0.099365234375, 1.03515625, 0.83447265625, 1.0107421875, 0.4833984375, 0.6806640625, -1.666015625, -1.701171875, -0.3935546875, 1.3876953125, -0.79248046875, -0.7099609375, -0.498046875, -0.8408203125, -0.1767578125, -0.431884765625, 0.53564453125, -1.576171875, -1.8095703125, 0.5556640625, -1.2333984375, -0.41650390625, -0.6533203125, -0.81005859375, -1.08203125, 0.1785888671875, 0.6318359375, -0.95263671875, 0.82373046875, 0.97265625, 0.2666015625, -0.32470703125, -0.428955078125, -1.365234375, -0.74462890625, 0.5869140625, -0.85498046875, 1.7587890625, -0.70849609375, 0.3125, -0.02935791015625, -1.8173828125, -0.4638671875, 2.13671875, 0.87158203125, 0.1529541015625, 1.28125, -0.8095703125, -0.60791015625, 1.8134765625, -0.6533203125, -1.875, 0.783203125, 0.57421875, -2.166015625, -1.1767578125, 1.083984375, 0.437744140625, -1.3134765625, -0.61376953125, 0.2509765625, -0.7822265625, 1.3603515625, -1.513671875, -0.2005615234375, -2.03515625, 1.2783203125, 1.072265625, 1.5, 0.2032470703125, 2.646484375, 0.41796875, -0.1343994140625, -0.1160888671875, 0.6015625, 1.8525390625, -0.90869140625, 0.53955078125, 0.95947265625, -1.6708984375, 0.568359375, -0.25732421875, 0.375, 0.591796875, -1.0595703125, -1.3330078125, 2.20703125, -0.9482421875, -0.40283203125, -0.9443359375, 0.1954345703125, 0.222412109375, -1.46484375, -0.469970703125, 0.390380859375, -1.3681640625, -0.8486328125, -0.2288818359375, 0.2296142578125, -0.130126953125, 1.458984375, -1.775390625, 0.580078125, 1.859375, 0.0977783203125, 1.7109375, 1.1015625, -0.7998046875, 0.281494140625, -0.29638671875, 1.751953125, 0.8994140625, 0.91455078125, 0.05926513671875, 0.35986328125, 0.349609375, 0.86181640625, -0.51318359375, 0.35791015625, 0.03814697265625, 0.055694580078125, 0.172607421875, 0.91455078125, 1.46875, -0.42822265625, -2.322265625, -1.1953125, -0.34912109375, 0.130615234375, -0.3330078125, -2.248046875, -0.78466796875, -1.1962890625, 0.78955078125, 0.78271484375, 0.37646484375, -1.4072265625, 0.77685546875, -0.00879669189453125, 0.5029296875, 0.7724609375, -1.4560546875, 0.10546875, -0.373046875, -1.9677734375, 0.2646484375, -0.20458984375, -0.51806640625, -1.0810546875, -0.37158203125, 2.27734375, 0.47119140625, -0.2452392578125, -1.349609375, -1.7685546875, -0.09228515625, 0.82080078125, -1.041015625, 0.1207275390625, -0.252685546875, 0.86865234375, 0.4453125, 0.5361328125, 0.92626953125, -1.68359375, -0.1552734375, 0.71044921875, -0.5126953125, -0.89111328125, -1.033203125, -0.0821533203125, 0.6943359375, 0.96240234375, -0.06365966796875, 0.998046875, 0.0036373138427734375, -1.70703125, -2.001953125, -0.90625, 0.0135040283203125, -1.8427734375, 1.36328125, 0.303466796875, -1.17578125, 1.1943359375, 0.1343994140625, 0.52880859375, 0.389892578125, 0.264892578125, -0.34228515625, 0.481201171875, -0.51171875, -1.44140625, 1.4814453125, -0.54296875, -0.451416015625, 0.7041015625, -1.158203125, 0.80810546875, 0.96484375, 0.5732421875, 0.931640625, -0.92431640625, -0.13134765625, 0.796875, -0.869140625, 0.84326171875, 0.026031494140625, -0.826171875, -0.4052734375, 0.08245849609375, 0.15283203125, 0.30810546875, 0.693359375, 0.2247314453125, 1.501953125, 0.73291015625, 1.330078125, 1.939453125, -0.2369384765625, -1.5791015625, 1.787109375, 1.3681640625, 0.1700439453125, 0.461181640625, 1.8046875, -0.6611328125, 0.352294921875, -0.427978515625, -0.13916015625, -0.599609375, 1.0283203125, -0.41064453125, -1.380859375, -0.86767578125, -1.04296875, -0.1488037109375, -0.6611328125, 1.619140625, 0.82470703125, 1.412109375, 0.6005859375, -0.7880859375, 0.92431640625, -1.5673828125, -0.379638671875, -0.2054443359375, -0.72265625, 0.513671875, -0.0648193359375, -0.7763671875, -0.64306640625, 1.2626953125, 1.4404296875, -2.52734375, -1.0517578125, 1.2802734375, -0.79150390625, 0.050750732421875, 1.36328125, -0.302734375, 0.32275390625, 0.406005859375, 1.5224609375, 0.84814453125, -2.208984375, -0.9072265625, 0.0592041015625, -1.8935546875, 0.253173828125, 0.52734375, -0.75439453125, 0.1649169921875, 0.7236328125, -0.7548828125, -0.25732421875, -0.283935546875, -2.580078125, -2.36328125, -0.0262451171875, -1.48046875, 1.46875, -0.95849609375, 0.86083984375, 0.83056640625, -0.00731658935546875, -0.5263671875, -1.7626953125, -1.6328125, -0.0247802734375, 0.6240234375, 1.3046875, 0.443603515625, 0.2269287109375, -1.099609375, -0.9658203125, -0.54296875, 0.11407470703125, 1.412109375, 1.0537109375, -0.93408203125, 1.232421875, -0.86328125, 0.4013671875, 0.5234375, -0.68408203125, 1.1806640625, 1.80859375, -0.947265625, 1.3974609375, -0.0016870498657226562, 1.0068359375, 0.76708984375, 0.352294921875, 0.9951171875, 1.818359375, -0.22119140625, 2.2265625, -1.8369140625, -1.2255859375, -0.27197265625, 2.259765625, 0.24853515625, -1.212890625, 0.95263671875, 0.82373046875, 0.27294921875, 0.0016956329345703125, 1.0908203125, -1.333984375, 2.09765625, -1.548828125, 0.71826171875, -0.11126708984375, -1.2138671875, 1.5517578125, -0.63671875, -0.498291015625, 0.2037353515625, 0.0265350341796875, -2.55078125, -1.59375, -0.32666015625, -0.277587890625, -0.326904296875, 0.77099609375, -1.248046875, -0.72412109375, -0.83154296875, -0.58203125, -0.043365478515625, -0.876953125, 0.75146484375, -0.32470703125, -0.055938720703125, -0.73974609375, -0.475341796875, -0.61279296875, 0.72314453125, 0.7568359375, 0.5205078125, 2.189453125, -1.7822265625, -0.59423828125, -0.1630859375, 0.63330078125, 1.126953125, 0.0003113746643066406, -0.453125, 0.2490234375, -1.6279296875, -0.787109375, 1.48046875, 1.4033203125, 0.5712890625, 1.1259765625, -0.2462158203125, 0.1781005859375, 0.139892578125, 0.5712890625, 0.63720703125, 1.4296875, -0.0784912109375, 0.98876953125, 0.49267578125, 0.58740234375, -1.3037109375, -0.55810546875, -0.92822265625, -2.359375, -0.1357421875, -0.076416015625, 0.13330078125, -1.408203125, -1.3779296875, -0.0804443359375, 0.755859375, 0.93798828125, 1.2275390625, 1.763671875, 0.78662109375, -0.7548828125, 1.6611328125, 0.787109375, -1.1943359375, -0.62158203125, 1.5087890625, 0.33642578125, 0.98583984375, -0.124267578125, 2.1171875, 0.8544921875, 0.57958984375, 0.8701171875, 2.330078125, -0.8408203125, -2.04296875, -0.041717529296875, -0.9267578125, 1.6826171875, -0.158447265625, 1.396484375, 0.8212890625, 1.6513671875, -0.521484375, -0.9111328125, -0.053009033203125, -1.27734375, -1.044921875, 0.1607666015625, 0.489990234375, -0.71826171875, 1.775390625, 0.0947265625, -0.8466796875, 0.04156494140625, 0.1204833984375, -0.1585693359375, 0.1300048828125, -0.51513671875, -0.38037109375, -0.00970458984375, -0.66357421875, 0.13623046875, 0.448974609375, -0.63818359375, 2.810546875, -0.68896484375, -0.8564453125, 0.26416015625, 0.1556396484375, -2.16015625, -0.43896484375, -1.0390625, -0.453369140625, -0.10235595703125, -0.0220489501953125, -2.685546875, -0.47216796875, 0.73388671875, 0.52294921875, -0.336181640625, 1.232421875, 0.8212890625, 0.1929931640625, 0.49853515625, 0.0212554931640625, 0.08148193359375, -0.94921875, -0.07891845703125, -1.1328125, -0.794921875, -1.3876953125, 0.394287109375, 1.3095703125, 0.1541748046875, -1.1787109375, 0.77490234375, -0.7216796875, -1.60546875, -0.32080078125, -0.45947265625, -0.08758544921875, 1.3955078125, 0.1614990234375, 0.9306640625, 0.1944580078125, 0.40966796875, 0.53857421875, 1.3056640625, 1.03125, 0.276611328125, 0.7060546875, -1.0302734375, -0.03173828125, 0.285888671875, 0.5654296875, 3.458984375, 0.42919921875, 0.151611328125, 1.921875, 0.38134765625, 0.3896484375, 0.6923828125, -0.369384765625, -0.6533203125, -2.552734375, -0.0257110595703125, -0.97509765625, 0.82470703125, -0.5322265625, -1.2822265625, -0.03338623046875, 1.3291015625, -1.234375, -0.7119140625, -0.11474609375, 0.10546875, 0.81591796875, -0.7236328125, 1.15234375, -0.97607421875, 0.5537109375, 0.43798828125, -0.39990234375, -1.609375, 0.69873046875, 0.826171875, 0.916015625, -0.042877197265625, 0.85595703125, -0.317626953125, -0.43310546875, -0.2474365234375, 0.3310546875, 0.7705078125, -0.7763671875, -0.385498046875, -0.2132568359375, -0.5791015625, -0.32763671875, 0.56201171875, 0.50439453125, 0.32861328125, 1.7783203125, -0.10821533203125, 0.455322265625, -0.43310546875, -0.006282806396484375, -1.0234375, -1.59375, -0.9990234375, 0.87939453125, -0.064697265625, -0.06939697265625, 0.37451171875, -1.5537109375, 0.470947265625, -0.52685546875, -0.430908203125, 0.01436614990234375, -0.724609375, -1.40625, -0.031707763671875, -1.0634765625, 0.01079559326171875, -0.7626953125, 0.79248046875, -0.390869140625, 0.07373046875, 0.1875, -0.5146484375, -0.488037109375, -0.392822265625, -0.7890625, 0.2081298828125, -1.884765625, 0.047149658203125, -0.257568359375, -0.099365234375, -0.4873046875, 1.78515625, 0.4892578125, -0.57958984375, -1.2880859375, -0.474365234375, -1.4150390625, 0.921875, -0.1693115234375, 0.239013671875, -0.28955078125, 0.56103515625, 0.611328125, 1.0029296875, 1.107421875, -0.50146484375, 1.08203125, 0.0401611328125, -0.145263671875, 0.06573486328125, -0.455322265625, -0.27783203125, -1.802734375, 1.7197265625, -0.66943359375, 0.41650390625, -0.383544921875, -0.311279296875, 0.8671875, 1.025390625, 2.7421875, -0.1688232421875, -0.252197265625, -0.998046875, -0.470458984375, 0.48193359375, 1.162109375, 0.53369140625, -1.232421875, 0.6318359375, 1.73828125, -0.371826171875, 1.7099609375, -0.38427734375, -0.0006623268127441406, -1.900390625, -0.89208984375, -1.47265625, -0.333740234375, 0.201171875, 0.19873046875, 1.087890625, 0.349609375, 1.091796875, -0.06915283203125, 1.2138671875, -2.166015625, 0.5966796875, -1.0263671875, -0.771484375, -0.7900390625, -1.3603515625, 0.3544921875, -0.7861328125, -0.58544921875, 2.27734375, -0.015411376953125, -0.045074462890625, -1.3095703125, -1.4716796875, -0.96240234375, -0.405029296875, 0.52490234375, -0.78466796875, -0.4921875, 1.0927734375, -0.404541015625, 0.630859375, 0.3486328125, 0.2093505859375, 1.2236328125, -0.11517333984375, -1.0068359375, -0.47265625, 0.0816650390625, 0.6240234375, 0.287841796875, 0.60302734375, -0.427978515625, -0.022247314453125, -1.640625, 1.201171875, -0.061279296875, -2.05859375, -1.1162109375, -0.7744140625, -0.241455078125, 0.334716796875, -1.380859375, 0.21826171875, 0.9892578125, -0.07275390625, 2.64453125, -1.7783203125, 0.8505859375, 0.49560546875, -0.66845703125, -2.330078125, 0.52294921875, 0.0044708251953125, -0.32373046875, 0.91650390625, -0.85888671875, -0.1868896484375, -2.08984375, -0.487548828125, -1.7275390625, 2.080078125, -0.51806640625, -2.703125, -0.09246826171875, -2.87109375, -1.4677734375, -1.3037109375, -0.038177490234375, 0.76220703125, 1.3701171875, 0.15576171875, -0.40380859375, 2.359375, -0.66259765625, 0.72265625, 0.07598876953125, 2.08984375, 1.025390625, -1.544921875, -1.0439453125, -1.7666015625, 0.63623046875, -1.0859375, 1.2119140625, 1.4833984375, -1.201171875, -0.548828125, -1.3330078125, -1.19140625, -0.3046875, 0.311279296875, 1.85546875, 1.5771484375, -0.08349609375, -0.51806640625, -1.515625, 0.06988525390625, -0.64697265625, -0.43701171875, -1.595703125, 0.298583984375, -0.0726318359375, 0.60498046875, 0.48828125, -1.2451171875, -1.1279296875, -2.927734375, -0.9296875, 0.382080078125, 0.231201171875, -1.0078125, 0.1572265625, 0.74462890625, 2.076171875, 0.2158203125, -1.4072265625, 0.5869140625, -0.447021484375, 3.212890625, 0.294189453125, 0.60400390625, -0.85205078125, 0.1964111328125, -0.763671875, 0.56103515625, -0.1494140625, 1.0146484375, 0.84814453125, -1.619140625, 0.48193359375, -0.1427001953125, -0.0001380443572998047, 0.53369140625, -0.697265625, 0.346923828125, -1.474609375, -0.12298583984375, 1.720703125, 0.8408203125, 0.0032787322998046875, 1.923828125, -1.0634765625, 1.7353515625, -1.70703125, -0.433837890625, -1.05078125, 0.77978515625, -0.6943359375, -0.0197906494140625, 0.195556640625, 2.22265625, -0.032989501953125, -0.07061767578125, -0.583984375, -0.04833984375, -1.7265625, 0.2274169921875, 0.1744384765625, -1.875, 0.6474609375, -1.2265625, -0.83349609375, 0.279541015625, 0.947265625, -0.0609130859375, -0.6533203125, -0.69091796875, -0.65576171875, 0.321044921875, 0.1258544921875, -0.176025390625, -0.5712890625, -1.4814453125, -1.9375, -0.302978515625, 0.300537109375, 1.126953125, -1.3984375, 1.4501953125, -0.85107421875, -0.2274169921875, 0.27734375, -1.5283203125, 0.406494140625, 0.05950927734375, 0.81689453125, 0.487060546875, -1.259765625, 0.2061767578125, 0.1490478515625, -1.310546875, -0.1697998046875, 0.10382080078125, -1.1376953125, 0.418212890625, -0.1328125, 0.12286376953125, 1.466796875, -1.0615234375, -0.303466796875, 3.00390625, 0.75830078125, 0.77587890625, 0.4326171875, -0.31396484375, 0.9111328125, 0.58154296875, 0.62158203125, 1.3994140625, 0.1510009765625, 0.179931640625, -0.429931640625, 0.39111328125, -1.4248046875, 0.5947265625, -0.19921875, -0.68310546875, -0.05804443359375, 0.42431640625, -0.10662841796875, 1.1845703125, -0.5283203125, -0.284423828125, -0.58154296875, -1.1123046875, 0.041473388671875, 1.0361328125, 0.58984375, 1.044921875, -0.2344970703125, 0.15380859375, 0.8525390625, 1.31640625, 1.607421875, -0.295166015625, 0.32666015625, 1.02734375, 2.205371856689453e-06, -0.2275390625, 0.42138671875, -0.85009765625, 2.048828125, 2.205078125, -0.3564453125, -0.73779296875, -2.19921875, -0.357177734375, 0.129150390625, 1.197265625, 0.1552734375, -0.84521484375, 1.2607421875, -0.7587890625, 1.3369140625, -1.212890625, 0.63671875, 0.201416015625, 1.0966796875, -1.78125, -1.10546875, 0.00019550323486328125, -0.45361328125, -0.2138671875, -0.40478515625, 0.806640625, 0.95947265625, 0.227783203125, -0.64794921875, 1.232421875, 0.51806640625, -0.76220703125, -0.270263671875, 0.90234375, 0.08746337890625, 1.865234375, 0.7333984375, 0.260986328125, 0.56005859375, 1.7890625, -0.10418701171875, 0.4716796875, 0.1802978515625, -0.1912841796875, 0.47412109375, -1.1650390625, -0.9423828125, -0.37646484375, 0.576171875, 0.07257080078125, -0.1651611328125, -0.79296875, -0.11004638671875, 0.521484375, -0.59912109375, -1.6142578125, 0.00727081298828125, -0.022674560546875, -0.51416015625, -0.185302734375, 1.1474609375, -1.203125, 0.99560546875, 0.23583984375, -1.6708984375, 0.15576171875, 0.66357421875, -0.5400390625, 0.2880859375, -0.4287109375, -0.88671875, 0.484375, 0.8994140625, -1.0263671875, 0.828125, -0.05535888671875, -0.60400390625, -0.9140625, 0.3017578125, -0.361572265625, -0.201171875, 1.28125, -0.8427734375, 3.12109375, -1.8056640625, -0.189453125, -0.049102783203125, -1.234375, 0.1895751953125, 0.38232421875, -1.103515625, -0.9619140625, 1.341796875, -1.923828125, 0.609375, 0.99169921875, 1.279296875, 0.05517578125, -1.3916015625, 0.572265625, -0.038909912109375, 1.4130859375, 0.017578125, -0.08441162109375, -0.400390625, -0.9541015625, -0.9443359375, 0.039825439453125, -0.0171966552734375, 0.489990234375, -0.437255859375, -1.0283203125, 0.95751953125, 0.783203125, 0.95458984375, 0.09844970703125, 0.87890625, 2.5390625, -1.169921875, 1.03515625, -0.1964111328125, -1.794921875, -0.9697265625, -1.224609375, -0.8896484375, -0.87890625, 1.3603515625, -0.1505126953125, -0.1514892578125, -0.439453125, 0.06695556640625, 1.6474609375, -0.050079345703125, 1.537109375, 1.5595703125, -0.66552734375, 0.556640625, -0.199462890625, -0.41259765625, -0.2042236328125, -1.677734375, -1.642578125, 0.17822265625, 0.313232421875, 2.271484375, -1.169921875, 1.2353515625, 0.46533203125, -0.02386474609375, 1.654296875, 0.406005859375, -1.8173828125, -0.8203125, -0.442626953125, 0.875, 1.0966796875, 0.5029296875, 0.240478515625, 0.4501953125, 0.82958984375, -0.36962890625, -0.5556640625, -0.332763671875, 0.2113037109375, -0.215087890625, 0.339111328125, -0.23388671875, -0.59375, 0.8359375, 0.4453125, 0.78369140625, -0.5380859375, 1.4873046875, 0.759765625, 0.4765625, 0.07818603515625, 2.97265625, -2.171875, -0.61328125, -0.0924072265625, 1.357421875, 0.06158447265625, 0.94287109375, 0.81689453125, 0.205810546875, -0.51513671875, 0.10455322265625, -0.0517578125, -0.52001953125, -0.5244140625, 0.37841796875, 0.66357421875, 0.68408203125, 0.1783447265625, 1.83984375, -0.5439453125, 0.76953125, 0.65576171875, 0.544921875, 0.515625, 0.0135650634765625, 0.548828125, 0.36767578125, -0.369384765625, 1.03125, -2.03515625, -0.83984375, -1.76171875, 0.043304443359375, -0.039947509765625, 1.16015625, -0.658203125, 1.53125, 0.03680419921875, -0.07623291015625, 0.89990234375, 0.525390625, 1.78125, -0.71826171875, 1.2626953125, -0.54052734375, 0.87255859375, 0.62548828125, -0.67578125, -0.270751953125, -0.03375244140625, -0.60302734375, 2.130859375, 1.1240234375, 0.337890625, -1.4951171875, -0.68505859375, -0.755859375, 1.16796875, -0.0340576171875, 0.66357421875, -0.3037109375, -0.458251953125, -0.46142578125, -1.12890625, 0.226318359375, 0.324462890625, 1.046875, 0.1849365234375, 0.54052734375, 0.22607421875, 0.51220703125, -0.8173828125, 0.1512451171875, 1.10546875, -0.095458984375, 1.9931640625, 0.485107421875, 0.330322265625, -0.74267578125, 0.88037109375, -1.0908203125, 1.8779296875, 0.98388671875, -2.408203125, 0.83935546875, 0.4453125, -0.143310546875, -0.3974609375, 0.052032470703125, 0.54443359375, 0.2392578125, -1.6650390625, 1.4208984375, 0.110595703125, -0.63671875, 1.55078125, -0.040924072265625, 0.759765625, 1.21875, -2.05078125, -1.12109375, 0.89453125, -1.1943359375, 0.54150390625, 0.437744140625, 0.99755859375, -0.274658203125, 2.580078125, 1.3349609375, 0.06610107421875, 1.1240234375, 0.962890625, 1.478515625, 1.388671875, -0.7138671875, -0.053436279296875, -0.415283203125, -0.8466796875, -0.677734375, -0.826171875, -1.3046875, -0.322509765625, -0.3330078125, -1.556640625, -0.1961669921875, -1.65625, -0.467041015625, 0.0204620361328125, 0.11871337890625, -0.18212890625, 0.4521484375, 0.5029296875, -0.970703125, -1.8310546875, 0.41015625, 0.92724609375, 1.1552734375, -0.6796875, -0.2607421875, 0.54296875, -0.61328125, 1.8544921875, 0.671875, -1.46484375, 0.488525390625, 0.38623046875, -0.2286376953125, 0.0181884765625, -0.1494140625, -2.41796875, 0.06549072265625, -1.09375, 1.56640625, -0.34130859375, -0.426513671875, -0.79248046875, 0.382568359375, 0.8671875, -1.091796875, -2.798828125, -1.078125, -0.1727294921875, -1.517578125, 0.146484375, -1.849609375, -2.865234375, -0.755859375, 0.9189453125, -0.036834716796875, 1.54296875, 0.1572265625, 0.84423828125, -1.6767578125, -0.320068359375, 0.170654296875, 1.0205078125, 0.88427734375, 0.55029296875, -0.4453125, -0.76513671875, 1.0205078125, 0.64794921875, 0.435791015625, 0.12115478515625, 1.45703125, -0.751953125, -0.288330078125, 2.78125, -0.95361328125, -0.548828125, -0.90869140625, -0.69677734375, -0.87939453125, -1.986328125, 1.1064453125, -0.12939453125, -0.96728515625, 0.41552734375, -0.15234375, -0.330078125, -1.0283203125, 1.0322265625, 0.89990234375, 1.271484375, 0.36474609375, -0.005939483642578125, -0.9140625, -0.83056640625, 2.224609375, 0.3759765625, 0.374267578125, -2.3515625, 1.0224609375, -1.0146484375, 0.443115234375, 0.63916015625, 0.61572265625, 0.65625, 2.283203125, -1.4677734375, 0.65966796875, 0.7431640625, 0.069091796875, -1.146484375, 0.99755859375, 1.474609375, -0.94189453125, -1.466796875, 0.0181884765625, -0.328857421875, -2.6796875, -2.03125, -0.1702880859375, -0.71044921875, 0.060943603515625, -0.2705078125, -1.34765625, -1.2275390625, -0.0300445556640625, -0.5927734375, 0.3017578125, -0.67431640625, 0.43359375, 0.50830078125, -0.1517333984375, 0.251708984375, 0.312744140625, -0.75927734375, 0.2386474609375, -1.3779296875, 0.55419921875, -1.0546875, 0.286376953125, 1.0625, -0.47412109375, 0.05828857421875, -0.463623046875, -1.490234375, -0.88330078125, 0.06976318359375, 0.460205078125, -0.97412109375, -0.9765625, 0.118896484375, 0.2220458984375, 0.84375, 0.951171875, 1.3642578125, -0.95361328125, -1.013671875, 1.5693359375, -0.59619140625, -0.529296875, -1.119140625, -1.8974609375, 1.310546875, -0.22802734375, 0.371337890625, 0.11163330078125, 0.8037109375, 0.1392822265625, 0.156005859375, 0.72021484375, 0.76416015625, 0.0450439453125, -0.822265625, 0.3359375, 0.5576171875, 0.625, -0.53564453125, -0.1868896484375, -1.5654296875, 0.65234375, -1.2275390625, 0.215087890625, 0.49560546875, 1.04296875, -0.1451416015625, 0.1348876953125, -0.474365234375, -0.169921875, -1.4384765625, 0.255615234375, -0.07843017578125, -2.22265625, 0.79052734375, -0.9072265625, 2.462890625, -0.82958984375, 1.1005859375, -1.61328125, 0.336181640625, -1.18359375, 0.8505859375, 0.326416015625, -0.48486328125, 0.5302734375, 0.72705078125, 1.134765625, -0.0367431640625, 0.0248260498046875, 2.443359375, -0.98388671875, 0.0709228515625, -2.826171875, -0.31005859375, 0.239501953125, -0.286865234375, -2.544921875, 0.7529296875, 1.6337890625, 0.036346435546875, 0.3046875, 2.431640625, 0.791015625, -0.153564453125, -0.6005859375, -0.77587890625, 0.08477783203125, 0.55908203125, 0.85888671875, -1.5712890625, -1.1630859375, 0.1470947265625, 1.91015625, -0.17236328125, 0.227294921875, -1.099609375, 1.6279296875, -0.61376953125, 1.4091796875, 0.427001953125, 0.271240234375, 0.7783203125, -0.1395263671875, -0.252197265625, 2.380859375, -1.810546875, -1.4990234375, 0.08184814453125, -1.9296875, 1.4853515625, 0.32275390625, 1.228515625, -0.76220703125, 0.7890625, -0.023529052734375, 1.802734375, 0.1856689453125, 0.92919921875, -0.49365234375, 0.316650390625, -0.62744140625, 1.240234375, -0.440673828125, -1.763671875, 0.425537109375, -0.465087890625, 0.80224609375, 0.04095458984375, -0.456298828125, 0.68603515625, 0.456298828125, 2.923828125, 1.1201171875, 0.1865234375, -0.322265625, 0.08599853515625, -0.2071533203125, 0.303955078125, -0.5849609375, 2.0234375, 0.0220184326171875, -0.5146484375, 0.12359619140625, 0.57470703125, -0.134521484375, 0.4423828125, -0.25732421875, -1.4619140625, 0.05133056640625, 0.3046875, 1.9228515625, -0.8720703125, 0.6015625, -0.57763671875, 1.203125, 1.3515625, 0.09136962890625, -1.8828125, 1.060546875, 3.177734375, 0.60693359375, -0.32470703125, 0.642578125, -1.099609375, -0.0213623046875, 0.11407470703125, -0.57080078125, 0.402099609375, 0.1534423828125, 0.72900390625, 0.64892578125, 1.3310546875, -0.6513671875, -1.46875, -0.1578369140625, 0.62939453125, -1.212890625, -0.50390625, 1.4892578125, 0.767578125, 0.127685546875, 0.794921875, 1.1630859375, 1.91015625, -0.65771484375, -0.31689453125, 1.0263671875, 0.79931640625, 0.30224609375, -0.0509033203125, 0.377685546875, 2.0390625, -0.79296875, 0.37158203125, 0.71826171875, -0.6708984375, 2.455078125, 1.423828125, 0.53369140625, 0.81982421875, 0.494140625, -1.9697265625, -0.5947265625, -0.81689453125, -1.130859375, -1.287109375, 0.08624267578125, 0.319091796875, 2.611328125, 2.28125, -1.0712890625, 0.11627197265625, 1.7705078125, -1.4384765625, 1.484375, -0.341064453125, -0.86865234375, -1.1376953125, -0.126953125, 0.14990234375, 0.296875, -1.271484375, 0.264404296875, -0.055023193359375, -0.59228515625, -1.3564453125, 0.030975341796875, -1.265625, 1.615234375, -0.11712646484375, -0.68701171875, -0.2227783203125, -0.11224365234375, -0.250732421875, 0.6318359375, 0.6279296875, 0.71875, -0.556640625, -0.93212890625, 0.2548828125, -0.66748046875, -0.265869140625, -0.304931640625, 0.491455078125, 1.7021484375, 1.3369140625, -0.63623046875, -0.2296142578125, -0.23046875, -0.127197265625, 1.216796875, -0.176513671875, 0.505859375, 0.1871337890625, -2.78515625, 0.76513671875, 1.5634765625, 0.6162109375, -0.76708984375, 1.201171875, -0.9443359375, 0.8564453125, 0.7998046875, -0.46728515625, -0.35302734375, 1.501953125, 0.0133819580078125, 0.96875, -0.6181640625, 1.126953125, 0.257080078125, 2.326171875, -0.7900390625, 1.23828125, 0.60888671875, -0.378173828125, -0.6591796875, -0.25390625, 0.515625, -0.1075439453125, -1.314453125, -1.845703125, -2.51953125, -0.58642578125, -0.89453125, -0.12646484375, -2.125, -0.96240234375, -0.05499267578125, 0.609375, -1.1728515625, 1.8076171875, 0.1505126953125, -0.7119140625, -0.053466796875, -0.1549072265625, -0.6953125, -0.89892578125, -0.3046875, -0.03302001953125, -0.7421875, -1.173828125, 0.31982421875, -1.42578125, -0.56884765625, 0.2476806640625, -0.056884765625, -0.5244140625, -0.63427734375, -1.3369140625, -0.082275390625, 0.1129150390625, 0.71826171875, -0.477294921875, 0.11474609375, -0.27880859375, 1.88671875, 0.68505859375, 0.2177734375, 0.69873046875, -0.32666015625, 0.42724609375, -0.393310546875, -1.6845703125, -0.0849609375, -0.5068359375, 0.57568359375, 0.287109375, 0.0296173095703125, 1.1982421875, -0.29736328125, -0.1298828125, -0.81982421875, 1.3427734375, 0.7587890625, -0.9345703125, 2.1796875, -0.92138671875, 0.262939453125, -0.146728515625, -0.64404296875, 0.191650390625, -1.552734375, 1.56640625, 0.9716796875, -1.837890625, -2.109375, 1.9208984375, 1.0224609375, -0.5830078125, -0.564453125, 0.86083984375, 1.166015625, -1.435546875, -0.284912109375, 1.4599609375, -1.2724609375, 0.12237548828125, 0.333251953125, -0.8671875, -0.6376953125, -0.037994384765625, -1.326171875, -0.8017578125, -1.388671875, -0.48046875, -0.36474609375, 1.423828125, 1.083984375, 0.52880859375, 0.432373046875, 1.6298828125, -1.0986328125, 0.2216796875, 0.71337890625, 1.09375, -0.90380859375, -0.9013671875, -1.1875, -0.552734375, 2.353515625, 1.591796875, -0.70458984375, -0.75244140625, -0.36669921875, 0.6884765625, -0.3095703125, 0.2381591796875, 0.6923828125, -0.5986328125, -1.3173828125, -0.634765625, 0.346435546875, 0.9462890625, 1.376953125, -0.7373046875, 0.85791015625, -0.1263427734375, -1.25, 0.53466796875, 0.06951904296875, -0.7314453125, -1.82421875, 0.6201171875, -0.306396484375, -1.701171875, 0.1524658203125, 1.388671875, 2.19921875, 0.4453125, -0.21923828125, 1.0703125, 1.248046875, -0.73388671875, -0.97705078125, 2.048828125, 0.81884765625, 0.2442626953125, 0.1788330078125, 1.041015625, 0.8271484375, 0.61669921875, -1.890625, 0.69775390625, -0.7568359375, 1.4609375, 0.21533203125, -0.560546875, 0.11895751953125, -0.1502685546875, 0.57177734375, 1.2509765625, 1.2861328125, -0.055023193359375, 1.818359375, 0.86083984375, -0.37939453125, 0.071044921875, 1.1064453125, -1.01171875, -2.70703125, -0.849609375, 0.82470703125, 0.78759765625, 1.6708984375, -1.8447265625, 0.450439453125, 0.1234130859375, 0.74462890625, 0.294189453125, -0.47802734375, 1.375, -0.5146484375, -0.4423828125, -0.27294921875, -0.361328125, -1.6826171875, -0.505859375, -1.1884765625, 0.923828125, 0.75048828125, -1.1318359375, -2.6328125, 0.04998779296875, 1.4384765625, 0.81494140625, -0.11956787109375, -0.451171875, 0.28564453125, 0.4521484375, -0.1256103515625, -0.92333984375, 1.5068359375, -0.513671875, -0.84521484375, -1.09375, -0.267333984375, -0.11614990234375, -0.76123046875, 0.055328369140625, -0.190673828125, 0.5771484375, 1.427734375, 0.6083984375, -0.389404296875, 1.6259765625, 0.49658203125, -0.837890625, 1.068359375, 0.0966796875, -0.947265625, -0.5830078125, -0.382080078125, 0.69287109375, 0.07330322265625, 0.80908203125, 0.8408203125, -0.451904296875, -0.48828125, -1.171875, 0.9423828125, -0.476806640625, -0.86572265625, 0.169677734375, 0.2880859375, -0.8974609375, 0.429931640625, 0.89306640625, -0.6748046875, -0.92236328125, -1.9638671875, -0.1778564453125, 0.1273193359375, -0.53466796875, -1.3720703125, -1.3046875, 0.60986328125, 0.736328125, 1.6962890625, -1.8935546875, 0.55859375, 1.5908203125, -1.931640625, -0.04058837890625, 2.3984375, -1.544921875, -1.1376953125, 1.3525390625, -0.56005859375, 0.5712890625, -0.0204925537109375, 0.6982421875, -1.650390625, -0.8720703125, -0.1961669921875, -0.5966796875, 1.0283203125, 1.91796875, -0.137451171875, 1.005859375, 0.0594482421875, -0.28515625, 0.476806640625, 0.180419921875, -1.7939453125, -1.349609375, 0.491943359375, -0.96240234375, 0.21142578125, 1.76953125, -0.5810546875, 0.307373046875, 0.023651123046875, -2.3125, 0.0211639404296875, 0.83740234375, 1.3251953125, -1.16796875, -1.2744140625, -0.72802734375, 0.0711669921875, 1.431640625, 0.06036376953125, -0.349609375, 0.78369140625, 0.02880859375, 0.426513671875, -0.061676025390625, 0.61328125, -0.80908203125, 0.13916015625, 1.376953125, 0.74853515625, 1.505859375, 1.279296875, 0.12890625, 1.2998046875, 1.1708984375, 1.8427734375, 0.25, 2.076171875, -0.091064453125, 1.9091796875, -1.4150390625, 0.04608154296875, 2.037109375, -0.75732421875, 0.60595703125, 1.1455078125, 0.74951171875, -0.17333984375, 0.50927734375, -0.9521484375, 1.0849609375, 0.1199951171875, 0.2371826171875, 1.900390625, -0.1256103515625, -0.184814453125, 1.115234375, -1.64453125, 0.1585693359375, 1.0, -0.232666015625, -1.1123046875, 0.69287109375, 1.923828125, 0.083251953125, 0.371337890625, 0.0762939453125, -1.7744140625, 0.038482666015625, 0.489990234375, 0.2310791015625, -0.5849609375, -0.448974609375, -0.040283203125, 0.59423828125, 0.260009765625, -0.669921875, 0.062347412109375, -0.56884765625, -1.3212890625, -1.1044921875, -0.1441650390625, -0.271484375, 0.376220703125, -2.162109375, -0.81201171875, 0.912109375, -0.65234375, -0.306640625, -0.091796875, -0.079833984375, 0.99658203125, 0.11309814453125, 0.53955078125, -0.254638671875, 1.4775390625, 0.258544921875, -0.413818359375, -1.9248046875, 0.156494140625, 0.689453125, -1.41796875, 0.266845703125, -0.023406982421875, 0.5078125, -1.1953125, -0.91015625, -0.26953125, -0.35791015625, 0.239501953125, 1.61328125, -0.8564453125, 0.427734375, -1.287109375, -1.2900390625, -0.331298828125, 1.029296875, 0.476318359375, 0.41064453125, 0.474365234375, -1.091796875, 1.474609375, -1.802734375, -0.64697265625, 0.54638671875, -0.50634765625, 0.1099853515625, -0.06219482421875, 1.2353515625, 0.93408203125, -1.3291015625, 0.8046875, -2.296875, -0.72900390625, 1.1796875, 0.6181640625, 0.345703125, 0.87060546875, -1.6533203125, -1.181640625, -2.72265625, -1.2841796875, 0.85546875, -1.1884765625, 0.23193359375, 0.54345703125, 0.479248046875, -0.55859375, 0.0555419921875, 0.08636474609375, -2.671875, 0.0989990234375, -1.46484375, -0.05401611328125, 1.12109375, 0.034271240234375, -1.951171875, -0.99609375, 0.398193359375, -1.3193359375, 0.207275390625, 0.043609619140625, -0.634765625, 0.93994140625, 0.286376953125, 1.63671875, 0.01457977294921875, 0.11566162109375, -0.88134765625, -0.52783203125, -0.6591796875, -0.1951904296875, 1.1943359375, 1.4248046875, 1.236328125, 0.319580078125, 0.097412109375, -1.51171875, 0.07098388671875, 0.0498046875, -0.344482421875, -1.0673828125, -1.0703125, 2.302734375, -0.230712890625, -2.59375, -1.50390625, 1.16796875, -0.04547119140625, -0.95068359375, -0.66650390625, -0.3134765625, -0.966796875, 0.427978515625, -0.76611328125, -0.317138671875, -0.5185546875, -0.77734375, 0.421630859375, -0.62353515625, 1.6533203125, -0.29541015625, -0.6865234375, 0.42919921875, -0.4326171875, 0.29345703125, -0.08599853515625, -1.1494140625, -1.0498046875, 0.347900390625, -0.042694091796875, 0.378662109375, 0.309814453125, -0.049560546875, -0.95068359375, -1.498046875, 0.4228515625, 1.7939453125, 0.09625244140625, 0.1781005859375, 0.29736328125, -0.2032470703125, 0.56103515625, 1.4033203125, 0.81396484375, -1.396484375, -0.6796875, -1.1962890625, -0.64990234375, -0.93212890625, 1.6767578125, 1.4140625, -0.38671875, 1.875, -1.5224609375, 2.02734375, 2.8046875, -0.71533203125, -0.1864013671875, -0.86181640625, -0.214111328125, 0.12158203125, -1.7373046875, 1.3251953125, 0.86669921875, 0.625, -0.1865234375, 0.326416015625, -0.198486328125, -0.492431640625, 0.431396484375, -1.2958984375, 0.642578125, 1.109375, 0.97216796875, 1.3505859375, 0.57373046875, -0.13720703125, 1.4375, -0.5517578125, 0.9287109375, 1.1708984375, -0.0836181640625, -1.9267578125, 0.583984375, 0.048065185546875, -0.399169921875, -0.397216796875, -0.043060302734375, -0.418701171875, 0.572265625, 0.240234375, 0.040435791015625, 0.4677734375, 1.1005859375, -0.40625, -0.84326171875, -1.32421875, -1.181640625, 0.430419921875, 1.5771484375, 0.77587890625, -0.48681640625, 1.5830078125, 0.0208587646484375, 0.94921875, -0.7666015625, -0.273681640625, 0.40673828125, 1.2353515625, -0.08343505859375, 0.487060546875, 1.4990234375, -0.12188720703125, 0.29345703125, 0.468017578125, 0.186767578125, 0.72998046875, -0.10296630859375, 0.1922607421875, -0.9775390625, -0.2413330078125, -0.249267578125, -1.2763671875, -0.1546630859375, 1.2470703125, 0.658203125, -1.3154296875, -0.8173828125, -0.4658203125, -1.001953125, -0.03131103515625, 0.91015625, 0.8291015625, 0.455078125, 0.11798095703125, -1.064453125, 1.4619140625, 1.486328125, -0.1636962890625, 0.06878662109375, -0.9013671875, 0.66455078125, 0.154052734375, 0.127197265625, 1.3623046875, -0.13232421875, 0.2413330078125, -0.87548828125, 0.103515625, 0.5615234375, -1.2373046875, -0.431640625, 1.01953125, -0.056396484375, 0.91796875, 0.548828125, 0.66748046875, 1.7861328125, -0.139404296875, 0.049591064453125, 0.56591796875, -0.36962890625, 1.873046875, 1.8740234375, 0.050567626953125, -1.7939453125, -0.290771484375, 0.58349609375, 0.3837890625, 0.153076171875, 1.19140625, 1.107421875, 0.94189453125, 0.30517578125, 0.71337890625, -0.57861328125, 1.08203125, 1.51953125, 0.194091796875, 0.4287109375, 1.666015625, 0.277587890625, 1.212890625, 2.052734375, 1.1337890625, 0.26025390625, -0.8935546875, -0.360107421875, -1.359375, -1.76953125, 0.77783203125, -2.392578125, -0.4541015625, 0.421630859375, 0.62109375, -0.57080078125, -0.61669921875, -0.0712890625, -0.03021240234375, -1.736328125, 1.3974609375, 0.77587890625, 0.86962890625, 0.32470703125, 0.681640625, 0.52783203125, 1.509765625, -1.2431640625, -0.119140625, 1.484375, -1.607421875, -1.396484375, 0.76904296875, -0.60009765625, -0.724609375, 0.31201171875, 1.5029296875, 2.076171875, -0.1732177734375, 1.8544921875, 1.2998046875, -0.038818359375, 1.2705078125, 0.39208984375, 1.4755859375, 0.391845703125, -0.615234375, 0.70361328125, -0.338623046875, 0.330078125, -0.67138671875, 0.04608154296875, 0.360107421875, -0.7587890625, -0.0275421142578125, -0.125244140625, -0.232421875, -0.8681640625, -0.1324462890625, -2.056640625, -0.125, 0.5810546875, 0.2030029296875, -0.82177734375, -1.7783203125, -0.4189453125, 0.794921875, -0.272216796875, -1.716796875, -0.36083984375, 2.015625, 0.74609375, 1.2275390625, 2.201171875, 1.2578125, -0.144775390625, -1.0361328125, 1.5830078125, -0.04559326171875, 0.476318359375, 0.5341796875, -0.376708984375, 0.689453125, 0.71923828125, -0.75, 0.003215789794921875, -0.85205078125, -0.51025390625, -0.08551025390625, -0.0266876220703125, 1.0205078125, 0.4755859375, 0.0190582275390625, 0.4619140625, 1.05859375, 0.86962890625, 0.505859375, -0.2213134765625, -0.52490234375, -1.1064453125, 0.259521484375, -1.6884765625, -0.377685546875, -1.33984375, 0.0182952880859375, -1.0537109375, -0.199462890625, -1.294921875, -1.2099609375, -0.48095703125, 0.31494140625, -0.787109375, 0.452880859375, 0.73828125, 0.95068359375, 1.1513671875, 1.671875, 1.2138671875, -0.1658935546875, -0.321044921875, -0.578125, -1.3544921875, -0.251220703125, -0.16259765625, 1.076171875, 1.078125, 0.298583984375, -0.3955078125, -0.89990234375, 0.478515625, 0.27294921875, 3.337890625, 0.345947265625, -0.4150390625, -0.1427001953125, 1.68359375, 1.2958984375, -1.3466796875, 1.1064453125, 0.0151214599609375, -1.3896484375, -0.7548828125, -0.444580078125, 1.9833984375, -0.16259765625, -0.30517578125, 0.66552734375, -1.001953125, 0.2333984375, -1.107421875, -0.023590087890625, -1.078125, -0.48486328125, -1.162109375, -1.521484375, -0.62890625, -0.2235107421875, 1.6240234375, 0.77880859375, 0.4345703125, -0.55517578125, 0.55419921875, -1.0, -0.2269287109375, 0.6298828125, 1.0615234375, 1.130859375, 0.1893310546875, 0.0196075439453125, -1.1044921875, 0.662109375, 0.4111328125, 0.3466796875, 1.1171875, -1.3798828125, 0.5634765625, 0.26416015625, 1.095703125, 1.2373046875, 0.06439208984375, 1.814453125, -0.001918792724609375, 0.43798828125, 1.2587890625, -0.5166015625, -3.001953125, 0.417236328125, -1.8876953125, -0.270263671875, -0.3115234375, -0.322265625, -0.2322998046875, 1.0849609375, -1.150390625, -0.93408203125, -0.267822265625, 0.60302734375, 0.024932861328125, -1.9453125, 1.501953125, -0.384521484375, -0.4921875, -0.75048828125, -1.4375, -0.36474609375, 0.0157928466796875, -1.0107421875, -0.2459716796875, -1.416015625, -1.1064453125, -1.04296875, 1.9189453125, -1.9482421875, 0.61767578125, 0.4375, 0.67529296875, -1.001953125, 0.28857421875, 1.3212890625, -0.40673828125, 0.70068359375, 1.845703125, -0.147705078125, -0.399658203125, 1.1962890625, -1.0810546875, 1.3798828125, 0.6015625, 0.1351318359375, -0.77880859375, 0.64111328125, -0.34765625, 0.7216796875, 0.1339111328125, -1.2333984375, 0.356201171875, 1.212890625, 1.8623046875, -0.8193359375, 1.76171875, 1.8466796875, -0.0355224609375, -0.8359375, 0.8212890625, 1.3486328125, 0.1429443359375, 0.356689453125, 0.94873046875, -1.46875, -0.2364501953125, 0.10589599609375, -0.26416015625, -0.5732421875, 0.2117919921875, 2.46484375, -0.51953125, -0.1629638671875, -1.1630859375, -0.77490234375, -0.80419921875, 2.041015625, 0.9716796875, 0.84423828125, -0.44091796875, -1.1396484375, -0.29541015625, -1.7490234375, 1.111328125, -0.1785888671875, -1.220703125, -0.84130859375, -0.90478515625, -0.1741943359375, 2.087890625, -0.55615234375, 1.3466796875, 0.87060546875, -0.2327880859375, -0.034332275390625, -1.3203125, 2.314453125, 0.2137451171875, 0.55078125, 1.0888671875, 0.6875, -0.1754150390625, -0.94384765625, -1.9189453125, -2.115234375, -0.312255859375, 0.79638671875, 2.423828125, 0.24267578125, 0.267333984375, 0.5107421875, -0.10101318359375, 0.5263671875, -0.68603515625, 0.4560546875, -0.4482421875, -0.1934814453125, -0.59814453125, -1.2119140625, -2.412109375, 0.94921875, 0.1585693359375, 0.296142578125, -1.2080078125, 0.7998046875, -0.64404296875, -0.3408203125, 1.318359375, 0.22802734375, -0.013824462890625, 0.1627197265625, -1.0166015625, -0.182861328125, 0.338134765625, 0.36962890625, 0.859375, 0.478759765625, -0.3154296875, -2.7734375, -1.22265625, 0.65380859375, 0.79248046875, 0.336181640625, -0.66796875, 0.7626953125, -1.048828125, -1.2060546875, -0.024200439453125, 1.3447265625, 0.056396484375, 0.1624755859375, -0.04949951171875, 1.146484375, -0.04986572265625, 0.62841796875, -1.359375, -1.072265625, 1.224609375, -1.6923828125, -0.447509765625, -0.454833984375, 1.0283203125, -1.6044921875, 0.389404296875, 0.8330078125, -0.7177734375, 1.208984375, -1.0595703125, 0.444091796875, 0.46533203125, -2.296875, 0.007076263427734375, -0.020782470703125, 0.42431640625, 2.619140625, -2.046875, 0.2269287109375, -2.671875, -0.94482421875, -0.07220458984375, 0.254150390625, -1.078125, -1.96875, -0.01207733154296875, -0.1956787109375, -0.331787109375, -0.2232666015625, -0.173583984375, -1.7548828125, 1.1689453125, -0.022216796875, 0.8427734375, 0.791015625, 1.2490234375, 1.52734375, -0.021209716796875, 0.374267578125, -0.7158203125, 0.98974609375, 0.414794921875, -0.11083984375, -0.744140625, 0.9853515625, 0.320556640625, 0.45556640625, -0.415283203125, 0.84716796875, 0.81005859375, -1.044921875, -1.197265625, 0.81689453125, 0.3798828125, -1.775390625, -1.40625, 1.5048828125, -1.5908203125, 1.18359375, 1.646484375, -0.509765625, 1.7109375, -0.1883544921875, 1.5458984375, -0.44677734375, 0.409912109375, 0.373291015625, -0.46337890625, 1.212890625, 0.5234375, 1.8291015625, -0.12396240234375, -1.248046875, 2.5, 0.2783203125, -0.8916015625, 0.051055908203125, -0.67626953125, 0.370361328125, 0.2303466796875, -2.30859375, -0.295166015625, -0.70068359375, 1.1318359375, 1.8271484375, -0.274169921875, -1.1259765625, 0.6376953125, 0.80224609375, 0.1729736328125, 0.86962890625, -0.7431640625, 1.40625, -0.37890625, 0.0231475830078125, -0.52880859375, 0.71484375, 0.9716796875, 0.8115234375, -1.8525390625, -1.359375, 0.198486328125, 0.464599609375, -0.004787445068359375, 0.370849609375, -0.29150390625, -0.06585693359375, -0.18701171875, 1.0947265625, -2.125, 0.184326171875, -0.8310546875, -1.5849609375, -1.353515625, 0.86328125, 0.2197265625, 0.0158538818359375, 0.72509765625, 0.93359375, 1.033203125, 1.12109375, -0.9638671875, -0.5732421875, -1.8798828125, -1.4619140625, 0.34814453125, 0.3955078125, -1.068359375, -0.4033203125, -0.164306640625, -0.2008056640625, -2.3515625, -0.5205078125, 0.68017578125, -0.418701171875, -0.7392578125, -0.7353515625, 1.2119140625, -0.2056884765625, 0.939453125, -1.431640625, -0.0255584716796875, 1.033203125, -0.0682373046875, 0.0096588134765625, 1.439453125, 0.57763671875, 0.60986328125, -0.7587890625, 1.037109375, -0.41845703125, -1.083984375, -0.59619140625, 0.0185699462890625, 0.007328033447265625, 0.1763916015625, 1.556640625, 0.74267578125, 1.2626953125, -0.1607666015625, 0.254638671875, 0.62841796875, -2.134765625, 0.243896484375, 2.01953125, 0.98193359375, -0.370849609375, 1.3818359375, 1.091796875, 0.20849609375, 0.48193359375, -0.79541015625, -1.587890625, 1.4638671875, 0.431884765625, 1.7861328125, 0.013153076171875, -0.1536865234375, 0.0880126953125, -2.115234375, 0.59326171875, 2.697265625, -1.4541015625, 1.1728515625, -0.3388671875, 0.81494140625, -0.5576171875, -1.5771484375, -0.88134765625, -1.0732421875, -0.2232666015625, 0.174072265625, -0.7119140625, 2.46875, 0.9765625, 0.55078125, -2.001953125, -1.662109375, -1.6796875, -0.459716796875, -1.1015625, -1.90625, 0.623046875, 2.431640625, -0.225830078125, -0.1719970703125, -0.309326171875, 1.25, 0.62109375, -0.476806640625, 2.64453125, 0.288818359375, 0.90283203125, 0.0017709732055664062, 0.1380615234375, 0.552734375, -1.49609375, 0.859375, 0.172119140625, 1.1650390625, 1.599609375, -0.33740234375, -0.38134765625, 0.011566162109375, 0.241943359375, -0.2408447265625, -0.1776123046875, -1.0146484375, -0.763671875, -0.85595703125, -1.3828125, -0.29833984375, -2.376953125, 0.203857421875, -0.517578125, -0.64404296875, -0.96923828125, -0.8681640625, -1.3876953125, 0.1729736328125, -1.0361328125, 1.435546875, -0.92333984375, -0.81298828125, -0.9970703125, 0.1165771484375, -1.5146484375, -0.08868408203125, -0.72265625, -2.15234375, 1.1181640625, 0.155029296875, 1.4697265625, -0.00791168212890625, 1.1982421875, -0.55078125, -0.77197265625, 0.94140625, 0.1341552734375, -0.884765625, 2.19140625, 0.1031494140625, 0.814453125, 1.2900390625, -1.7841796875, 0.01113128662109375, 1.05078125, 1.6337890625, -0.6220703125, -0.7548828125, -0.0791015625, -0.67041015625, -0.4453125, -0.2841796875, 0.430419921875, 0.7314453125, 1.060546875, 0.5390625, 1.0751953125, 0.2548828125, -2.271484375, 0.650390625, -0.1424560546875, 0.5615234375, -0.93701171875, 0.857421875, -0.42041015625, 0.4990234375, 0.650390625, 0.329833984375, -0.712890625, -0.046905517578125, 0.8701171875, -0.013763427734375, -0.191162109375, -0.7705078125, -0.87841796875, -0.348388671875, 0.7587890625, -0.61376953125, 0.488525390625, -0.78125, 0.323974609375, 0.481201171875, -1.244140625, 0.73486328125, -0.59326171875, 0.09063720703125, -1.166015625, 2.052734375, -0.076904296875, 1.06640625, 1.423828125, -0.73828125, -0.1539306640625, -1.3955078125, 1.2607421875, -0.494384765625, 0.28271484375, 0.1107177734375, -0.64111328125, -0.309814453125, 1.33984375, 0.52392578125, 0.67626953125, -1.09375, 0.97265625, 0.1732177734375, -1.6640625, 1.5009765625, -0.87353515625, -2.578125, -0.9111328125, -0.90234375, -0.285888671875, -0.429931640625, 2.08984375, 0.1436767578125, -0.6650390625, 0.6865234375, 0.15185546875, -0.609375, 0.396728515625, 0.497802734375, -0.07427978515625, 0.74755859375, -0.8447265625, 0.70556640625, 0.564453125, 0.1650390625, 1.115234375, 1.6845703125, 1.42578125, 1.0712890625, -1.3212890625, -1.1728515625, -0.228271484375, 1.2021484375, 0.9541015625, -0.0238494873046875, -1.349609375, 0.411865234375, 0.2509765625, 0.86376953125, -1.181640625, 0.58544921875, 0.494873046875, -0.53466796875, -1.5751953125, -2.04296875, 0.16259765625, -1.7744140625, 0.029754638671875, 0.75732421875, 0.71533203125, 0.44921875, -0.051788330078125, -0.275634765625, -0.48486328125, 2.20703125, -0.2054443359375, 0.7119140625, 0.73046875, 0.94580078125, 1.322265625, 1.29296875, 1.103515625, -0.44482421875, -0.039306640625, 1.0224609375, -0.94970703125, 0.1943359375, 0.0709228515625, -0.10626220703125, -1.4306640625, -0.81982421875, -1.517578125, -0.66650390625, 2.07421875, -1.0087890625, -0.2122802734375, -0.89794921875, -0.9619140625, 1.2939453125, -0.78076171875, 0.640625, 0.703125, -1.5390625, 1.3193359375, -0.50439453125, -0.1871337890625, 2.025390625, 0.62353515625, -0.89013671875, 0.82763671875, -1.205078125, -1.8828125, -0.548828125, 0.8359375, 0.47998046875, -1.0400390625, -0.35791015625, 0.041595458984375, -3.119140625, 0.70849609375, 1.0185546875, -1.33203125, 0.430419921875, 0.447998046875, 0.0401611328125, 0.80810546875, -1.779296875, 1.533203125, -0.6591796875, -1.447265625, 0.8583984375, 2.138671875, 0.2100830078125, 0.3046875, 0.62353515625, 0.07440185546875, 0.6455078125, 0.056976318359375, -0.2099609375, 0.179443359375, -0.18603515625, 0.454345703125, -0.07855224609375, 0.392822265625, -0.11328125, 0.05120849609375, -1.0263671875, -0.071044921875, -0.1378173828125, -0.0576171875, -0.36962890625, -0.136474609375, -1.287109375, 2.1953125, 0.755859375, -0.316162109375, 1.4873046875, 0.9111328125, -1.412109375, 0.226318359375, -0.2509765625, 1.447265625, 0.51513671875, 0.86767578125, 1.5087890625, -1.0302734375, 1.1640625, -0.1600341796875, 1.18359375, 1.6279296875, -0.74072265625, -1.1259765625, 0.68701171875, 0.44287109375, 0.85498046875, 0.396728515625, -1.1630859375, -1.0712890625, -0.1072998046875, -1.353515625, -1.580078125, 0.05584716796875, 0.201416015625, -0.428466796875, -0.837890625, 0.72802734375, 0.2144775390625, 0.046478271484375, -0.544921875, -0.254638671875, -1.5517578125, 0.273193359375, -0.422607421875, -0.333740234375, 0.65380859375, 0.32177734375, -0.9580078125, 0.7373046875, 0.07928466796875, -1.0654296875, -0.9150390625, 1.4267578125, 0.01396942138671875, -0.06964111328125, -0.08953857421875, -0.267578125, 0.478515625, -1.046875, 1.3974609375, 0.50341796875, 0.72119140625, 0.34130859375, -1.3154296875, -0.8056640625, 0.97705078125, 0.016998291015625, 0.10040283203125, -0.293701171875, -2.642578125, -1.490234375, 1.08984375, -0.52490234375, 1.2197265625, -0.4951171875, -2.0390625, -0.06011962890625, -0.7158203125, -1.5712890625, -0.180908203125, 0.058013916015625, -1.5029296875, 0.66455078125, 0.9609375, 1.26171875, 0.287353515625, 0.9912109375, -1.1591796875, -0.031585693359375, 0.461181640625, 0.58203125, -1.0927734375, 1.70703125, -0.223388671875, 0.97705078125, 0.9716796875, 1.4365234375, 0.214599609375, 0.1138916015625, -1.0126953125, 1.716796875, 0.153564453125, 1.478515625, -0.8515625, -2.408203125, 2.701171875, 1.140625, -0.0743408203125, 0.62158203125, 1.41796875, 0.978515625, -1.94140625, 1.1953125, 0.1873779296875, -0.47705078125, -0.6435546875, 0.90234375, -1.6787109375, -1.3798828125, -0.293212890625, -0.314208984375, -2.0859375, 0.10797119140625, -1.3701171875, -0.537109375, -0.2008056640625, -0.90771484375, 0.2088623046875, 1.001953125, 0.52978515625, -0.14794921875, -0.94287109375, 0.40380859375, -0.47607421875, -0.09002685546875, 1.689453125, 0.5498046875, 0.1376953125, 0.91064453125, 1.6533203125, -0.180908203125, -0.45361328125, -1.46875, 1.62109375, 2.212890625, -0.239990234375, 0.53271484375, 1.1689453125, 0.57275390625, -0.90283203125, -1.4912109375, -0.82666015625, 0.1439208984375, -0.0660400390625, 1.6279296875, 0.0689697265625, -1.025390625, -0.63623046875, -1.2099609375, 1.837890625, -0.71826171875, -1.6171875, -1.0625, -0.56787109375, 0.356689453125, 0.359130859375, -1.3447265625, 0.54541015625, -0.09637451171875, 0.794921875, 0.394775390625, -0.06011962890625, -1.4248046875, 1.4130859375, 1.91015625, -0.2418212890625, -0.39306640625, 0.63623046875, 0.9111328125, -1.3671875, -2.220703125, 1.4091796875, 1.279296875, -1.486328125, -0.0254974365234375, -1.1376953125, -0.701171875, -0.292236328125, -0.5927734375, -0.91650390625, -0.9912109375, 1.8056640625, -2.25390625, -1.216796875, 0.039703369140625, 2.517578125, 0.8955078125, 0.90478515625, 0.429931640625, 0.70166015625, 0.338623046875, -0.309814453125, 0.984375, 0.094482421875, 0.1888427734375, 1.5966796875, -1.2490234375, 0.09539794921875, -0.55615234375, 0.80712890625, -1.1875, 0.365234375, -0.60400390625, 2.05859375, -0.0755615234375, -1.2802734375, -1.525390625, 0.123779296875, 1.6181640625, -0.31201171875, 0.8828125, 1.8349609375, -0.70458984375, -1.3369140625, 0.24462890625, -1.5126953125, 0.80712890625, -0.3203125, -0.67529296875, -0.2025146484375, -0.30517578125, -1.326171875, -0.260498046875, -0.1781005859375, -1.2958984375, 1.4072265625, -1.154296875, -0.479248046875, 0.83349609375, 0.461669921875, -1.36328125, -1.6474609375, 0.7890625, -2.244140625, 0.4443359375, -1.83984375, -0.646484375, 0.12432861328125, 0.159423828125, 0.43994140625, -0.365234375, 0.1048583984375, -0.35693359375, 1.7802734375, -0.005405426025390625, 0.85595703125, -0.7265625, 0.2042236328125, 0.302490234375, -0.294921875, -2.162109375, 0.552734375, -0.146484375, 1.98046875, 0.461669921875, -0.362548828125, -1.0322265625, -0.8701171875, -0.6708984375, -0.509765625, -0.6103515625, -0.57666015625, -0.716796875, -0.7412109375, -0.66259765625, -0.478759765625, 0.334228515625, 0.80126953125, -0.62158203125, 1.5078125, -1.27734375, -0.401611328125, -1.2236328125, 0.58984375, 0.439453125, 0.208251953125, 0.52978515625, -2.1328125, -0.10394287109375, 0.4013671875, -1.0732421875, 1.1123046875, -0.1617431640625, -0.515625, -0.34814453125, -0.8173828125, 1.0888671875, -1.880859375, 0.89697265625, -0.25927734375, -1.5751953125, -0.9990234375, -2.27734375, 0.58935546875, -1.3916015625, 0.7900390625, -1.1279296875, 0.69384765625, 0.76806640625, -0.398681640625, -0.671875, -0.5732421875, 0.0222625732421875, 1.216796875, -0.0103912353515625, -0.2032470703125, 0.765625, -0.1485595703125, 0.25390625, 0.85693359375, -0.8818359375, 1.3134765625, -0.70068359375, -0.2293701171875, -0.478271484375, -1.419921875, -1.740234375, 1.033203125, 0.9111328125, 0.75830078125, 1.7841796875, -1.7021484375, -0.6259765625, -1.3623046875, 0.306396484375, 1.5146484375, -0.58935546875, 1.703125, 0.86865234375, 0.3720703125, -2.1796875, 0.01418304443359375, -1.7041015625, 0.525390625, -0.39794921875, -0.9697265625, -0.316162109375, 0.67431640625, 0.68408203125, -1.4853515625, 1.3232421875, -0.77294921875, 0.378662109375, 0.253173828125, -0.88037109375, -0.8017578125, 0.470458984375, 0.85107421875, 0.173828125, 1.107421875, -0.2113037109375, 1.044921875, 0.1163330078125, -0.437744140625, -1.830078125, -1.166015625, 1.4033203125, 0.72314453125, -0.088623046875, 0.52783203125, 0.84912109375, -1.0732421875, 0.8369140625, 1.3623046875, 0.29443359375, 0.280517578125, 0.008758544921875, 0.2286376953125, 0.52783203125, 0.408935546875, 2.505859375, -0.33203125, 0.281494140625, 1.1044921875, -0.5654296875, -0.212890625, 0.9404296875, -1.2392578125, 1.6435546875, -0.0352783203125, -1.271484375, -1.228515625, 1.39453125, 0.197509765625, -0.75537109375, -1.38671875, -0.95166015625, 0.0175018310546875, 0.93212890625, 0.85888671875, -1.52734375, -1.0712890625, 0.79736328125, -0.40478515625, 1.3828125, 0.2841796875, 0.56787109375, 1.267578125, -0.2236328125, -2.177734375, 1.0791015625, 0.83837890625, -0.290771484375, -0.483154296875, 0.2325439453125, -0.27783203125, 0.2406005859375, 0.6357421875, -1.794921875, -0.0165557861328125, 0.5419921875, -0.2236328125, -1.0615234375, -0.5478515625, 0.7177734375, -0.384521484375, -0.69140625, -0.0731201171875, 0.288818359375, -1.96875, 0.220458984375, 1.8212890625, 0.51220703125, -0.06402587890625, 1.6337890625, 0.1644287109375, 0.6826171875, 0.5634765625, -0.5458984375, -0.7841796875, 0.435302734375, -0.66796875, -0.89599609375, 0.0821533203125, 0.1334228515625, -0.72802734375, 1.40234375, -0.86279296875, 0.016998291015625, -0.58642578125, -0.0631103515625, 0.5048828125, 2.001953125, 0.763671875, -0.03363037109375, 0.6220703125, -2.224609375, 1.8994140625, -0.88916015625, -0.7431640625, 0.05908203125, -0.314453125, 0.62744140625, -2.298828125, -1.7421875, 0.27783203125, 1.8447265625, 0.356689453125, -0.25244140625, 0.029876708984375, 0.365478515625, -1.6142578125, 1.083984375, 0.12158203125, 0.388916015625, -0.459228515625, 0.471435546875, 0.49609375, 0.237548828125, -0.513671875, 0.2303466796875, -0.39990234375, 0.003131866455078125, -1.0517578125, 0.3388671875, -1.9326171875, 1.0556640625, -1.2587890625, -0.080078125, -0.388916015625, -1.8544921875, -1.05078125, -0.39990234375, -1.1904296875, -0.56201171875, -0.0819091796875, 0.238037109375, 1.1767578125, -0.57958984375, -0.37744140625, -1.234375, 1.2041015625, 0.2484130859375, -0.57958984375, -1.7451171875, -1.8388671875, -0.1309814453125, 0.8486328125, 0.01715087890625, 1.841796875, 1.0908203125, 0.132080078125, -1.3583984375, -0.69873046875, 1.798828125, 0.0933837890625, 0.403076171875, 1.2490234375, 0.322509765625, -0.72509765625, -0.73095703125, -0.09832763671875, -0.14404296875, -0.07733154296875, -1.1748046875, -1.818359375, 0.5751953125, 0.314697265625, 0.9296875, 1.0283203125, -0.0565185546875, -0.491943359375, -0.359619140625, -0.7607421875, 1.07421875, 0.90966796875, -0.187255859375, -0.97705078125, 1.4423828125, -0.05548095703125, -1.853515625, 0.472412109375, -0.1331787109375, 1.0546875, -0.1624755859375, -0.71337890625, -0.466552734375, 2.09375, -1.2802734375, -1.423828125, 1.234375, 0.1270751953125, -0.1749267578125, 0.439453125, -0.09307861328125, -0.33154296875, 0.81689453125, 0.49169921875, -0.67578125, -0.2091064453125, 0.630859375, -0.3623046875, -1.6767578125, -1.5283203125, -1.2958984375, 1.1982421875, 1.244140625, -0.2174072265625, 0.10308837890625, 0.66064453125, -0.1363525390625, 0.467041015625, -0.2054443359375, -0.0123138427734375, -0.74462890625, -0.25, 1.7275390625, -0.327392578125, 1.966796875, -0.52197265625, -0.8427734375, -0.91796875, 0.71630859375, -0.033538818359375, 0.430908203125, 0.258056640625, 0.5810546875, 0.85986328125, 0.2037353515625, -0.9541015625, -2.66796875, -0.1539306640625, 2.5703125, -1.390625, -0.9912109375, -2.92578125, 0.1207275390625, -0.1258544921875, 1.12890625, 0.65771484375, 0.314208984375, 0.30615234375, -1.6396484375, -0.246337890625, 0.70654296875, -0.040252685546875, 0.7744140625, -1.37109375, -0.129638671875, -0.892578125, 0.486083984375, -1.18359375, 0.4619140625, -1.025390625, -0.7490234375, 1.2802734375, 1.552734375, -0.0303497314453125, -0.06353759765625, -0.442626953125, 2.484375, 0.67529296875, -0.2080078125, -1.96484375, 0.796875, -0.0733642578125, -1.2734375, 0.1346435546875, -0.487060546875, -0.98974609375, -0.455322265625, -1.0068359375, -0.23486328125, 2.4375, -1.4169921875, 0.7099609375, -1.4228515625, 0.6435546875, 0.8193359375, 0.90283203125, -1.07421875, -0.1953125, 1.0419921875, 0.164794921875, -1.955078125, -0.323486328125, -0.14794921875, -0.09783935546875, 0.93408203125, 0.68994140625, 0.11163330078125, -1.373046875, -2.044921875, -0.01480865478515625, -0.52197265625, 0.392578125, 0.338623046875, 1.3798828125, 0.0169830322265625, -1.201171875, -0.212158203125, 1.64453125, 0.59765625, -1.681640625, -0.258544921875, -1.5625, 0.127197265625, 0.364013671875, 0.425048828125, 1.1806640625, 0.1036376953125, 0.69140625, 1.9287109375, -1.0830078125, 1.982421875, 0.46484375, -1.5390625, 1.8837890625, -1.5771484375, -0.0782470703125, -1.212890625, -0.73291015625, 0.755859375, -0.19873046875, 0.994140625, -0.2156982421875, 0.2626953125, -0.312744140625, -0.0653076171875, -2.3125, -0.151123046875, -0.638671875, -0.416748046875, -0.73974609375, -0.85107421875, -0.55810546875, -0.7421875, -1.2626953125, -1.42578125, 1.2001953125, -0.164794921875, 0.82177734375, 0.8095703125, 1.4140625, -2.2890625, 0.279541015625, 0.90185546875, -0.56787109375, 3.025390625, 0.2022705078125, 0.228271484375, -0.85009765625, 1.0341796875, -1.513671875, -1.1435546875, 1.103515625, 0.08563232421875, 0.03509521484375, -0.58349609375, 0.2156982421875, 0.62548828125, 0.58740234375, -0.254150390625, 1.0087890625, -0.580078125, 0.49951171875, -0.70703125, -0.5849609375, 0.269775390625, -1.2109375, -0.3369140625, -1.2099609375, -0.88720703125, 0.69189453125, -0.85546875, -1.09375, -1.3974609375, -0.544921875, 0.008453369140625, 0.72705078125, 0.2489013671875, 0.050079345703125, -1.525390625, -0.5361328125, -0.053619384765625, 1.572265625, 0.759765625, 1.5234375, -0.98828125, 2.126953125, -1.7548828125, -1.234375, 1.013671875, -0.2099609375, -1.521484375, -1.70703125, -0.379150390625, 2.21875, 0.857421875, -2.251953125, -1.0615234375, 0.0726318359375, -1.4072265625, -0.97705078125, -2.078125, -0.124755859375, 0.2159423828125, 1.6337890625, -0.1602783203125, 0.6083984375, -0.1678466796875, -1.294921875, 0.279296875, -0.5595703125, 0.442138671875, 0.456787109375, -0.110107421875, -1.330078125, -1.8408203125, 0.8115234375, -0.10711669921875, 0.98876953125, 0.70263671875, -0.367919921875, 0.481689453125, 0.058013916015625, 0.72314453125, 1.4501953125, 1.4677734375, -1.53125, -0.701171875, 1.8046875, 0.280517578125, -0.49462890625, 0.450439453125, -1.0751953125, 0.9736328125, -1.0791015625, 0.65625, -0.97265625, 0.7109375, -0.63671875, -1.2109375, -0.2265625, -0.395751953125, 0.88427734375, 1.58203125, -0.84619140625, -0.454345703125, 0.771484375, 2.060546875, 0.1275634765625, -0.79052734375, -2.4921875, -0.303466796875, 1.1435546875, 0.77099609375, 0.97021484375, -0.68994140625, -1.3193359375, 0.81689453125, 0.67578125, 0.269775390625, 1.560546875, -0.003063201904296875, 0.259765625, 1.2421875, -0.908203125, 1.2451171875, -1.1865234375, 0.031402587890625, 1.1220703125, 0.128173828125, -0.16064453125, -0.1614990234375, 0.6640625, 0.3046875, -0.8564453125, -0.0168304443359375, -0.90478515625, -0.2353515625, -0.0697021484375, 0.33642578125, -0.326904296875, -0.8740234375, 0.49560546875, 0.5048828125, 0.0848388671875, -0.9970703125, -0.98974609375, -0.8037109375, -1.8408203125, 0.6494140625, 1.15625, 0.329345703125, 0.4365234375, 0.11956787109375, 0.344482421875, -0.51123046875, 1.080078125, 0.236328125, 1.037109375, 1.1240234375, -1.701171875, -0.578125, 0.2388916015625, 0.349365234375, 1.2177734375, -0.810546875, -0.37353515625, -0.267578125, 0.71240234375, 1.3017578125, -0.8232421875, -0.395263671875, 2.36328125, 0.82470703125, 0.248779296875, -2.048828125, -1.4384765625, 0.06439208984375, 1.4609375, -0.71142578125, -1.46875, -0.37109375, 1.0166015625, -0.95361328125, 0.05584716796875, -1.7392578125, 0.24560546875, 0.80322265625, 0.69873046875, 1.046875, 0.35498046875, 1.19140625, 0.307861328125, 1.033203125, -0.76123046875, 0.3388671875, 0.35205078125, 1.1591796875, -0.77734375, -0.46044921875, -0.387451171875, -0.947265625, -1.57421875, -1.2939453125, -0.34033203125, -0.63330078125, 0.1533203125, -1.2939453125, 0.2086181640625, 1.1005859375, -0.376708984375, -0.0792236328125, 1.5390625, -0.441162109375, -1.310546875, -0.385009765625, 1.916015625, -2.3515625, 0.6025390625, 1.9091796875, -0.52294921875, 0.5, -0.146728515625, 1.05859375, 0.712890625, -0.32275390625, 0.40380859375, -0.15966796875, 0.58154296875, -0.0523681640625, 0.1484375, -0.751953125, 0.403564453125, -1.1005859375, -0.2919921875, 0.267822265625, 0.334228515625, 0.46337890625, 1.248046875, 1.3076171875, 0.0260467529296875, -0.66455078125, 0.1934814453125, -1.431640625, -0.572265625, -1.0244140625, 0.21533203125, -0.619140625, 0.57861328125, 0.63232421875, -1.5087890625, -1.45703125, 0.8193359375, 0.826171875, 1.01953125, 0.59814453125, 1.373046875, -1.8544921875, 0.385498046875, 0.317138671875, 0.056365966796875, -0.970703125, -0.1529541015625, -0.301513671875, -0.463134765625, 3.466796875, -0.233154296875, 0.1351318359375, 0.2188720703125, 0.05230712890625, -0.6474609375, 0.407958984375, -1.236328125, -1.4384765625, -0.10791015625, -1.1279296875, 0.470947265625, -0.7353515625, -0.355712890625, -0.267333984375, -0.7236328125, 0.2423095703125, -0.5654296875, -1.5458984375, -0.270751953125, -0.9638671875, -1.568359375, -0.8046875, -1.2626953125, -0.69287109375, 0.60498046875, 0.8349609375, 1.134765625, -0.10870361328125, -0.5947265625, 0.2244873046875, -1.029296875, 1.3876953125, 0.429443359375, -0.62841796875, -1.99609375, 0.2354736328125, 1.3701171875, 0.67041015625, -0.469482421875, 0.2220458984375, -0.473876953125, -0.0341796875, -0.6220703125, 1.8203125, -0.0775146484375, 1.7177734375, -0.66259765625, 0.83935546875, 1.380859375, 0.342529296875, 0.35400390625, -0.208984375, -1.67578125, -1.873046875, 0.2685546875, 1.8349609375, 0.12420654296875, -0.1513671875, -0.34912109375, -0.69970703125, 0.49365234375, 0.1712646484375, -0.055328369140625, -1.3798828125, -0.849609375, -1.3310546875, -1.111328125, -0.982421875, 0.49560546875, -1.36328125, 0.490234375, 0.17529296875, 0.93505859375, -1.4794921875, -0.2109375, -1.2216796875, 0.6357421875, -0.394775390625, 0.0987548828125, -0.198974609375, -0.88427734375, 0.144287109375, 0.650390625, 0.2078857421875, -0.430908203125, 0.87255859375, 1.408203125, 0.45458984375, -1.19921875, 0.59228515625, 0.7470703125, 0.369873046875, 0.452392578125, -0.378662109375, -1.6123046875, -0.051116943359375, -0.050323486328125, 0.388671875, -1.1103515625, 0.038116455078125, 0.89453125, -1.673828125, 0.89013671875, -0.024749755859375, -0.9091796875, -0.3603515625, -0.6923828125, 0.927734375, -0.4794921875, 0.54638671875, -1.181640625, 1.2421875, -0.69580078125, 0.33154296875, -0.830078125, 0.0474853515625, -0.1546630859375, -0.583984375, -0.392822265625, -1.904296875, 0.60888671875, 1.2978515625, 0.54345703125, -0.46875, -0.40673828125, -0.183349609375, 2.8359375, -1.4423828125, 0.89697265625, 1.5087890625, 0.07550048828125, -0.7841796875, -0.207763671875, -0.378173828125, 0.1285400390625, 0.83056640625, 0.59912109375, -1.8857421875, -0.01412200927734375, -0.70654296875, 0.10296630859375, 0.84326171875, 0.07818603515625, 1.90234375, -1.3701171875, -2.060546875, 0.43994140625, -1.083984375, 1.1376953125, -0.384033203125, -0.380126953125, -0.2822265625, 0.57421875, 0.401611328125, 0.89453125, 1.701171875, -0.359375, -0.7021484375, 0.8388671875, 1.142578125, -1.5751953125, 0.31591796875, 0.416259765625, 0.412353515625, 0.623046875, 0.37841796875, -0.0787353515625, 0.54248046875, 0.466064453125, -0.72509765625, 0.057159423828125, -0.12384033203125, -0.01016998291015625, -0.453857421875, -0.1378173828125, 0.364990234375, -0.5283203125, -0.417236328125, -3.232421875, -0.275634765625, 1.55078125, -0.7607421875, 1.6376953125, 0.51171875, 1.2060546875, 0.0286865234375, -0.548828125, 1.3603515625, 0.11749267578125, -0.00661468505859375, -0.64892578125, -2.578125, -0.408935546875, -0.0151519775390625, -0.68115234375, 1.4150390625, -0.188720703125, -0.054779052734375, 0.37060546875, -1.48046875, -0.412841796875, 0.31298828125, -0.54931640625, 0.1536865234375, 1.1142578125, 1.404296875, -1.7841796875, -0.0628662109375, 1.873046875, 0.2313232421875, -0.043304443359375, 0.004383087158203125, 0.0180816650390625, -0.9697265625, -0.85302734375, -0.9833984375, -0.765625, 1.1845703125, -1.2001953125, -0.7099609375, -1.2578125, -1.8408203125, -1.3056640625, -0.7939453125, 0.15478515625, -1.23828125, -0.603515625, 0.80322265625, 1.328125, -0.023956298828125, -0.306640625, 0.99169921875, -1.4921875, -0.91064453125, -0.68994140625, -0.56396484375, -0.57373046875, 1.11328125, -0.1324462890625, 0.373779296875, -0.1373291015625, -0.3056640625, 0.005828857421875, -1.1572265625, 0.8505859375, 0.318115234375, 0.43603515625, 0.9814453125, -0.1380615234375, -0.283203125, -2.37890625, -0.007171630859375, 0.193603515625, -0.396240234375, -0.0191192626953125, 0.1126708984375, -1.4052734375, -0.435302734375, 0.282470703125, 0.425048828125, -0.7880859375, -1.7275390625, 1.3056640625, -1.142578125, -0.734375, 1.9775390625, -0.513671875, -1.951171875, -1.3681640625, 0.461181640625, -0.249755859375, 0.79248046875, -1.2138671875, -1.0390625, -0.411865234375, -0.90380859375, 0.98486328125, 0.2080078125, -0.40087890625, 0.0016078948974609375, -1.373046875, 0.481201171875, 1.7890625, -1.3212890625, -0.56201171875, 0.76904296875, 0.81884765625, -0.93310546875, -1.1962890625, -1.4375, 0.908203125, -1.28125, -0.09893798828125, -0.2320556640625, 1.111328125, -0.00450897216796875, -0.56982421875, 1.720703125, 1.3544921875, -0.76318359375, 0.916015625, -0.5361328125, -1.4228515625, 0.75048828125, 0.609375, 0.0010814666748046875, 0.0860595703125, 0.27880859375, 0.951171875, -0.4453125, 0.443115234375, -0.07562255859375, 0.2108154296875, 2.57421875, 0.572265625, -0.9072265625, -0.49658203125, 1.5693359375, 0.42724609375, 1.359375, -0.5888671875, 0.339111328125, 0.08355712890625, -0.1883544921875, -0.12359619140625, 0.80517578125, 1.2392578125, 0.473388671875, 0.8515625, 0.64453125, -0.048553466796875, 2.287109375, 0.388671875, 0.90625, -0.56884765625, 0.78662109375, -2.224609375, 1.2353515625, 0.366943359375, 2.099609375, 0.68212890625, 0.81005859375, -0.5986328125, 1.41015625, -0.31787109375, -0.82861328125, -0.1405029296875, -1.62890625, 1.09375, 1.6123046875, -0.81689453125, -0.5390625, -1.0869140625, -0.6025390625, -0.94775390625, 0.297119140625, -1.8740234375, -0.740234375, 0.75244140625, -0.31201171875, -0.1531982421875, -1.44140625, 0.4306640625, 1.15234375, 0.50146484375, 0.59033203125, 1.6396484375, -1.3642578125, -0.299560546875, -2.50390625, 0.03857421875, 0.064208984375, 0.480224609375, -0.2919921875, 0.335693359375, 0.0243377685546875, -0.210693359375, 0.0294647216796875, 0.360595703125, 0.73193359375, 0.7451171875, -1.9990234375, -0.1856689453125, -0.45068359375, 0.452392578125, 0.072021484375, -0.1114501953125, -0.5478515625, 1.9287109375, -1.4921875, -0.25146484375, -0.094970703125, 0.12744140625, -0.0928955078125, -0.60009765625, 0.07830810546875, 0.325927734375, -0.708984375, -0.6044921875, 0.444091796875, -0.61669921875, 0.06494140625, 0.97314453125, 0.64453125, 0.10150146484375, -0.498779296875, -0.57275390625, 0.30859375, -1.1640625, 1.1767578125, -0.716796875, -0.62060546875, -1.7060546875, -0.54443359375, 0.2054443359375, -0.2802734375, 0.0012884140014648438, -0.6962890625, 1.216796875, 0.5224609375, 0.059173583984375, -0.37646484375, -1.8076171875, 1.0869140625, -1.6884765625, 0.27099609375, 1.14453125, -1.05078125, 0.442138671875, 1.7255859375, -0.006244659423828125, -0.8828125, -0.353759765625, 0.130615234375, -0.0037517547607421875, 1.30078125, 1.38671875, -1.263671875, -0.216064453125, 0.72509765625, 1.095703125, 0.1256103515625, -0.322509765625, 0.77685546875, -0.0416259765625, 0.54150390625, -0.79736328125, -1.5869140625, -2.3984375, -0.08984375, -0.23876953125, -3.380859375, -0.0467529296875, 1.10546875, -0.389404296875, 0.9775390625, -0.1328125, -1.1689453125, -2.41015625, 0.21142578125, -0.437255859375, -0.50341796875, -0.46337890625, -1.150390625, -0.309326171875, 0.287841796875, 2.166015625, 0.396240234375, 1.69140625, 0.0367431640625, -0.2088623046875, 0.02471923828125, -1.2373046875, 0.25048828125, -0.1041259765625, 2.251953125, 1.427734375, -0.70947265625, -0.45556640625, -1.1982421875, -0.81298828125, 1.953125, -0.326416015625, -1.3427734375, 1.1845703125, -0.48388671875, 1.5439453125, 2.28515625, 1.3740234375, 0.369384765625, -1.0673828125, 1.048828125, -0.485107421875, -0.60498046875, 0.68310546875, 0.5, -0.207763671875, -0.94580078125, -0.1610107421875, -1.28125, -1.4609375, 0.1697998046875, 1.322265625, -0.68212890625, -0.38525390625, -1.0927734375, 0.381103515625, 1.7353515625, -0.37744140625, 1.140625, 1.802734375, -2.98828125, 0.0297393798828125, -0.58203125, 0.06292724609375, -0.0484619140625, 1.658203125, 2.005859375, -1.9833984375, -0.6103515625, -1.1328125, 0.9931640625, -1.1591796875, 0.428955078125, -1.5830078125, -0.6904296875, -1.38671875, 0.36474609375, 0.35546875, -0.11920166015625, -0.269287109375, -0.450439453125, -0.166259765625, -0.658203125, 1.220703125, -1.61328125, -0.66455078125, 2.216796875, -0.1536865234375, -0.298095703125, 0.1036376953125, -0.097412109375, -0.65185546875, -0.5791015625, 0.7734375, -1.4072265625, 0.5166015625, -0.74462890625, -1.0703125, -0.2939453125, -0.03656005859375, 1.24609375, 0.78369140625, 0.81494140625, -0.03594970703125, 2.111328125, 0.689453125, 0.407958984375, -0.0017595291137695312, -0.2008056640625, 0.4609375, 0.26416015625, -0.2218017578125, -0.433837890625, -0.75439453125, 0.68115234375, -0.1572265625, 0.359375, 0.6259765625, -0.82861328125, -0.5576171875, 0.8544921875, -0.322021484375, 0.301513671875, -0.71337890625, -0.91943359375, -0.38427734375, 0.0804443359375, -0.341552734375, 1.2568359375, 0.896484375, 0.281005859375, 1.453125, -0.0218658447265625, -0.75, -0.455810546875, -0.330810546875, -0.52001953125, -0.53662109375, 0.6103515625, -0.1689453125, 2.029296875, -0.140625, 1.185546875, -1.091796875, -0.053802490234375, 0.4033203125, -0.477783203125, -0.95166015625, 0.9970703125, 0.415283203125, 0.8525390625, 1.2978515625, -0.043365478515625, 0.33642578125, 0.44140625, -0.10443115234375, 0.476806640625, -1.173828125, -1.2275390625, -2.46484375, 2.04296875, -0.262451171875, 0.266357421875, 0.6318359375, -0.1173095703125, -0.326171875, -1.0693359375, -0.488525390625, -1.2724609375, -0.414794921875, 1.892578125, 1.06640625, -2.0546875, -0.2071533203125, 0.136474609375, -0.1920166015625, 0.062347412109375, 1.6103515625, -0.6787109375, 0.6767578125, 0.623046875, 0.9599609375, -0.51318359375, -2.2734375, -0.247314453125, -0.931640625, -0.35107421875, 0.88232421875, 0.30419921875, -0.86376953125, -0.007595062255859375, -0.064453125, 0.448974609375, 0.455322265625, -0.228271484375, -0.31201171875, 0.57861328125, 1.435546875, 1.15234375, 1.037109375, 1.2265625, 0.0770263671875, 0.255615234375, -1.5615234375, -1.033203125, 0.99267578125, -0.22216796875, 0.8349609375, -2.3359375, 0.65576171875, -1.8564453125, 0.83447265625, 1.3037109375, 0.326171875, -0.249755859375, -1.408203125, -0.888671875, -0.457763671875, -0.7998046875, 0.273681640625, -0.51318359375, -1.025390625, -0.076416015625, -1.130859375, -0.556640625, 0.308349609375, 2.029296875, -0.376953125, -1.728515625, -0.275146484375, 1.037109375, -0.3974609375, 0.09405517578125, -0.71826171875, -0.79443359375, 0.30859375, -0.78955078125, 1.09375, 0.43115234375, 0.5439453125, 0.91845703125, 0.55322265625, -0.161376953125, 0.92431640625, -1.1376953125, 0.474365234375, 1.3505859375, 1.93359375, -1.208984375, 0.95166015625, -0.52294921875, -0.11224365234375, -0.71630859375, 0.99951171875, 0.69775390625, 1.091796875, 0.80126953125, 0.413330078125, -0.330810546875, 0.5166015625, 0.56201171875, -0.28662109375, -0.495849609375, -0.95166015625, 0.2364501953125, -1.6005859375, -0.27978515625, -0.37353515625, 0.4267578125, -0.2666015625, 0.79296875, -0.07012939453125, 1.1650390625, -1.087890625, -1.6064453125, -1.1376953125, -0.296142578125, 0.79345703125, 0.470458984375, -0.407958984375, -0.03265380859375, -0.3876953125, 0.433837890625, 0.00743865966796875, -1.814453125, 1.28125, -0.246826171875, 0.053192138671875, 1.205078125, 0.68408203125, -1.4482421875, -0.2357177734375, 1.732421875, 0.1629638671875, 2.580078125, -1.4794921875, -0.2274169921875, 1.296875, 1.45703125, 1.8515625, -0.353759765625, 1.40234375, -0.10455322265625, 0.54541015625, -0.765625, 0.46240234375, -0.00806427001953125, -2.197265625, -0.7578125, -0.1551513671875, 0.9921875, -0.458251953125, -1.4599609375, -0.38671875, -0.37548828125, -1.7275390625, -0.416259765625, 0.73095703125, 0.44189453125, 0.43701171875, -0.9189453125, 0.7158203125, -2.06640625, 0.018402099609375, -1.4658203125, -0.86328125, 0.08734130859375, -1.7890625, -0.974609375, 0.57861328125, -1.0712890625, -1.6953125, 0.271728515625, -0.1068115234375, 0.59130859375, 0.319091796875, 0.98095703125, -1.767578125, -1.71484375, -0.9326171875, 0.6142578125, 0.4150390625, -0.471923828125, 1.482421875, 1.3525390625, -1.4775390625, -1.3837890625, 0.253662109375, 0.85546875, -0.2242431640625, -0.6689453125, 0.157470703125, 1.0068359375, 1.203125, 0.48486328125, 0.4580078125, -1.4052734375, -0.47705078125, -1.3037109375, 0.137939453125, -0.92578125, 0.61328125, -0.1014404296875, 0.916015625, 0.55712890625, 0.576171875, 0.264892578125, 1.322265625, -0.9228515625, -0.493896484375, -0.73681640625, 0.041473388671875, 0.61865234375, -0.034027099609375, -0.283203125, 1.7490234375, -0.85107421875, -1.4541015625, -0.68701171875, -0.60400390625, 0.1444091796875, 2.244140625, 1.5419921875, 0.63427734375, 0.281982421875, -1.0517578125, -1.9560546875, -0.53857421875, -0.328125, -2.04296875, -0.2337646484375, 1.4267578125, 0.71826171875, -0.84228515625, -1.291015625, 0.62646484375, -0.0399169921875, -1.0234375, -0.28662109375, -0.6083984375, 0.56982421875, -0.306640625, 0.7255859375, 0.50341796875, 0.6611328125, -1.4892578125, -0.75, 2.57421875, -0.26806640625, 0.482421875, 0.9404296875, -1.791015625, 1.4619140625, -1.18359375, -0.52294921875, 1.21484375, -1.685546875, 0.366943359375, -1.009765625, 0.96240234375, 1.3525390625, -1.1015625, -1.083984375, 0.52392578125, 0.9775390625, -0.228759765625, 0.73828125, -0.11334228515625, 2.853515625, 1.5712890625, 1.306640625, -1.044921875, 0.76611328125, 0.05194091796875, 0.009033203125, 0.8515625, -2.150390625, -1.919921875, 1.5771484375, 0.359130859375, 0.5234375, 1.1787109375, 0.736328125, 1.2470703125, 0.78759765625, 0.345458984375, 0.282470703125, 0.400390625, -0.30078125, 0.126708984375, 0.5341796875, -0.01090240478515625, -0.107421875, -0.41943359375, -0.67626953125, -1.0029296875, -0.0187530517578125, 0.401123046875, 0.039337158203125, 0.119384765625, 1.0595703125, 1.3515625, -0.64990234375, -1.6416015625, -0.76708984375, 1.5087890625, -1.3623046875, -1.1240234375, -1.654296875, -2.5078125, 1.91796875, -1.1484375, 1.0283203125, 0.9814453125, -0.93017578125, 0.3154296875, -0.262451171875, -1.828125, -0.78857421875, -0.40380859375, 1.322265625, -0.002117156982421875, -0.382568359375, -1.1064453125, -1.982421875, -2.27734375, 0.59375, -1.1181640625, -0.8125, -0.3583984375, 0.1552734375, 1.56640625, 0.212890625, -0.447998046875, -0.7734375, 0.849609375, 1.203125, 0.50830078125, -0.58935546875, -0.56640625, 0.8779296875, -1.12109375, -0.1805419921875, -0.131103515625, -1.2275390625, -0.52392578125, -0.41015625, -0.74072265625, 1.4814453125, 0.412353515625, 0.61962890625, 1.3076171875, 0.888671875, 0.9814453125, -0.250244140625, 0.9853515625, -0.34375, -1.3974609375, 0.927734375, 1.1318359375, 0.25634765625, -0.69140625, 1.85546875, 0.09521484375, -0.452880859375, -0.8291015625, 0.09112548828125, -0.379638671875, -0.8720703125, -0.00806427001953125, 2.69140625, -0.2529296875, -0.05804443359375, -0.98046875, 0.8759765625, 1.9228515625, 0.71142578125, 1.291015625, -0.7255859375, 1.24609375, 0.50927734375, -0.496826171875, -0.159423828125, -1.6337890625, 0.69873046875, -0.65771484375, -1.0361328125, -0.229248046875, -0.81591796875, 0.86083984375, -0.25732421875, -0.732421875, 2.107421875, 0.9560546875, -0.247314453125, -0.359375, 0.223876953125, 0.105224609375, -0.170654296875, -1.498046875, 0.399658203125, -0.66455078125, -1.5478515625, -0.51171875, 0.61328125, 0.296142578125, 0.034088134765625, 1.5302734375, 1.814453125, 0.58837890625, 0.420166015625, 0.67041015625, -0.34423828125, 2.228515625, -0.62451171875, 0.89404296875, -1.4052734375, -1.2255859375, 0.340576171875, -0.49267578125, 0.6015625, -1.138671875, -0.72509765625, -1.10546875, 0.2083740234375, -0.1748046875, 0.890625, -0.5126953125, 1.0244140625, 0.78369140625, -0.4794921875, 0.08062744140625, -2.396484375, 4.750490188598633e-05, 0.5810546875, -0.79443359375, -0.2239990234375, 0.7998046875, -1.013671875, 0.08343505859375, -0.341796875, -1.2158203125, -0.6103515625, 0.97216796875, -0.880859375, 0.477783203125, 1.318359375, 1.251953125, -0.2283935546875, -1.7666015625, 0.51171875, -0.09722900390625, 0.7060546875, -0.2890625, 1.423828125, -0.327392578125, -2.525390625, -1.56640625, 0.60107421875, -0.82568359375, -0.5244140625, 0.53125, 0.196044921875, 0.236572265625, -0.2578125, 1.45703125, -0.75732421875, -1.583984375, 0.54150390625, -0.4423828125, 0.8955078125, -0.56298828125, 0.54931640625, 0.1474609375, 2.373046875, 0.2763671875, 1.060546875, -0.1485595703125, -0.57421875, 0.456298828125, -0.8134765625, 0.53466796875, -1.00390625, -0.85400390625, 0.9619140625, -0.73779296875, -0.587890625, 0.28271484375, -0.34326171875, -1.0830078125, -0.95458984375, -0.56884765625, 1.4013671875, -0.41455078125, 0.288330078125, 1.05078125, -1.3720703125, 0.7041015625, -0.625, -0.9736328125, -0.6611328125, -0.07501220703125, -0.5009765625, -1.546875, -0.345458984375, 0.470947265625, 1.3095703125, -0.56591796875, -0.019866943359375, -0.11553955078125, -0.95263671875, 1.5390625, -0.94189453125, 0.79443359375, 0.301513671875, 2.15234375, -0.80859375, 0.74560546875, -0.84375, -1.7021484375, -0.365966796875, 0.96728515625, 2.171875, 0.017578125, -0.65771484375, 1.474609375, -0.53759765625, 1.171875, 0.254638671875, -1.107421875, -0.40380859375, 0.74267578125, -1.634765625, -0.7861328125, 1.20703125, 0.222900390625, -1.6123046875, 0.271240234375, -0.5322265625, 1.1845703125, 0.2269287109375, -1.068359375, -2.37890625, 1.03515625, 2.296875, 1.3779296875, -0.96484375, -0.379150390625, -0.410400390625, 0.7744140625, 0.330078125, -1.77734375, 1.20703125, -0.84814453125, 0.51416015625, 0.2188720703125, 0.10650634765625, -0.60546875, 1.4599609375, 0.394287109375, -1.328125, -0.18505859375, -1.58984375, 0.54833984375, 0.367431640625, 0.6484375, 1.9306640625, -1.4951171875, 0.97412109375, -1.4638671875, 1.119140625, 0.50732421875, 0.9677734375, 0.669921875, -1.63671875, -0.71240234375, 0.4189453125, 0.86279296875, 0.1593017578125, -1.330078125, -1.0322265625, 0.703125, -0.32568359375, -0.2303466796875, -1.0087890625, -1.7666015625, -0.328125, -0.1573486328125, 0.4287109375, 1.2216796875, -1.748046875, -0.23486328125, -0.58740234375, -0.76171875, 1.03125, -0.86669921875, 0.60009765625, -0.1884765625, 0.377685546875, 0.33251953125, -0.377685546875, -0.0182647705078125, 1.2919921875, 1.095703125, 0.31494140625, -1.16015625, -2.5625, 0.42236328125, 0.163330078125, 0.06756591796875, 0.53369140625, -0.46826171875, 2.080078125, -1.0634765625, -0.58203125, -1.525390625, -0.94287109375, -0.11590576171875, 1.8837890625, -1.076171875, -0.2452392578125, -0.8037109375, 1.6650390625, 0.356201171875, 0.775390625, -1.1826171875, -0.354736328125, -0.3330078125, 0.0960693359375, -1.8212890625, -0.71337890625, -1.1396484375, -1.68359375, 0.351806640625, 0.4169921875, -1.6328125, -1.392578125, 1.02734375, 0.6298828125, 0.424560546875, -0.030853271484375, 0.57275390625, -0.055999755859375, 1.09765625, 0.158203125, -0.420166015625, 0.466064453125, -0.1851806640625, 1.6474609375, -0.043060302734375, 1.2138671875, -0.17822265625, 0.19921875, -1.2685546875, 1.30859375, -0.339111328125, -0.57763671875, 0.0361328125, -0.50341796875, 0.74365234375, -0.865234375, 0.0516357421875, 0.1949462890625, -1.4794921875, -3.1171875, 1.6943359375, -2.18359375, -1.1982421875, -0.75244140625, -0.7939453125, 0.2156982421875, -0.7021484375, -1.1552734375, 0.1834716796875, 1.33203125, 1.990234375, -1.845703125, 0.298583984375, 1.0615234375, -0.9267578125, -1.7626953125, 1.162109375, 0.413330078125, -0.334228515625, 0.80029296875, 0.028228759765625, -1.21875, -0.5859375, 1.3359375, 0.98095703125, 2.22265625, 0.68359375, 1.3037109375, 1.3037109375, -0.328857421875, 0.042694091796875, 0.86767578125, 0.050537109375, -0.8740234375, 2.0859375, -0.4677734375, -1.9951171875, -0.2442626953125, -1.2529296875, 1.4560546875, -0.7216796875, 0.84814453125, 0.70458984375, -0.75537109375, -0.2149658203125, -0.72265625, -0.46435546875, -0.484375, 0.49609375, -1.806640625, 0.51220703125, 0.363037109375, 0.25634765625, -1.4033203125, 0.8916015625, -0.640625, 0.8046875, -1.6005859375, 0.0775146484375, 0.310302734375, 1.517578125, -0.51513671875, -1.63671875, 0.483642578125, 0.192626953125, -0.2113037109375, 1.14453125, -0.384033203125, 0.167724609375, 1.15234375, -0.6064453125, 0.8095703125, -0.1317138671875, 1.0205078125, -0.0115203857421875, 0.23583984375, -1.1845703125, 0.5517578125, 0.55029296875, 0.52880859375, 1.619140625, 0.07049560546875, -0.51318359375, 0.62744140625, -0.455322265625, -0.56005859375, 0.86279296875, -0.89208984375, 0.70947265625, 0.0513916015625, 0.5322265625, 1.5439453125, -0.7255859375, -1.05078125, -1.1162109375, 0.318603515625, -3.025390625, -0.642578125, 0.0355224609375, 0.85888671875, 0.052459716796875, -0.22119140625, -0.1351318359375, -0.372314453125, 0.56396484375, 0.0313720703125, 0.304443359375, -0.28857421875, 1.5078125, -0.2822265625, -1.4814453125, -1.0751953125, 0.3388671875, 0.065185546875, -0.66357421875, 0.59033203125, 0.00444793701171875, -1.728515625, -0.193359375, -0.07525634765625, 0.446533203125, -0.75927734375, -1.5927734375, -1.0888671875, -1.03515625, 0.447998046875, 1.0869140625, -0.112060546875, -0.810546875, 0.28076171875, -0.60888671875, -0.9609375, 1.1884765625, 0.1622314453125, 1.5341796875, -0.5068359375, -0.33447265625, 0.3310546875, -0.2264404296875, 1.3515625, 0.1566162109375, -0.35595703125, 0.89111328125, -0.927734375, -1.943359375, 2.34765625, 1.1025390625, 0.1405029296875, 1.4462890625, -1.4306640625, -1.0595703125, 1.7646484375, 0.9599609375, 1.212890625, 0.55615234375, 0.55029296875, -0.69091796875, 0.57421875, 0.93994140625, -0.28466796875, 0.3662109375, -0.826171875, -1.185546875, 0.2108154296875, 0.1497802734375, -0.64990234375, -0.798828125, 1.076171875, -1.5546875, -0.412841796875, 1.703125, -1.1650390625, -0.46484375, 0.79833984375, -0.05126953125, 0.71044921875, -0.08734130859375, 0.348388671875, 0.409912109375, 0.33349609375, -1.4091796875, 0.587890625, -0.60546875, -2.251953125, 2.07421875, -0.7587890625, -1.5517578125, -2.0390625, 1.685546875, -2.19921875, -0.346435546875, 0.89111328125, 0.247314453125, 0.78759765625, -1.0751953125, -0.84716796875, -1.10546875, -0.423583984375, 0.2318115234375, 1.8525390625, -0.8251953125, 0.29248046875, -0.251953125, 0.64599609375, -0.039398193359375, -1.6904296875, -0.32763671875, 0.9931640625, -0.4736328125, 0.12005615234375, 1.7236328125, -1.2998046875, 0.369140625, 1.474609375, -0.2171630859375, 1.078125, -0.2802734375, 0.9208984375, -1.12890625, 0.955078125, 0.8095703125, 0.2276611328125, -0.12493896484375, -0.185791015625, 1.376953125, 1.16015625, 1.0556640625, -0.50830078125, -0.266845703125, 0.74560546875, -1.048828125, -1.546875, 0.259033203125, -0.5087890625, -0.153564453125, -0.177978515625, 0.2276611328125, -0.389892578125, -0.116943359375, -1.3037109375, 0.1259765625, 1.2548828125, -0.051422119140625, -1.4892578125, 0.63037109375, -1.271484375, 1.4462890625, 1.3720703125, -0.1978759765625, 1.484375, 1.0478515625, 0.0166168212890625, -0.0214996337890625, 0.6630859375, 0.32275390625, 0.5224609375, 0.99267578125, 1.0546875, -0.1048583984375, -0.258544921875, 0.38037109375, -0.11407470703125, 1.2412109375, -0.49365234375, 0.7763671875, -0.34033203125, 1.4482421875, 0.5166015625, 0.24853515625, 0.461669921875, -0.481201171875, 1.064453125, -1.29296875, -0.295166015625, -0.333251953125, 1.5390625, -0.330810546875, 0.035919189453125, 0.71826171875, -1.0810546875, 0.09613037109375, 0.72509765625, -0.5234375, -1.103515625, -0.59912109375, 0.061004638671875, -2.068359375, -0.11962890625, 0.78173828125, -0.424560546875, -1.251953125, 0.8251953125, 0.7607421875, 0.96044921875, -0.069091796875, -0.8779296875, -0.37939453125, -0.319091796875, 1.09765625, 0.3388671875, 0.2149658203125, -0.05865478515625, 0.5693359375, 0.01284027099609375, -0.64892578125, 0.50537109375, -1.0283203125, -0.11846923828125, -0.97509765625, 1.88671875, -1.7802734375, -1.162109375, -0.149169921875, 1.603515625, 1.109375, 0.35791015625, -0.8212890625, 0.419677734375, -0.61767578125, -1.763671875, 0.07550048828125, -0.0030364990234375, -0.35888671875, 0.276611328125, -0.043121337890625, 0.466064453125, 1.197265625, -0.82421875, 0.292724609375, -0.79736328125, -0.2330322265625, 0.10009765625, -0.465576171875, 0.141845703125, 0.2587890625, 1.2880859375, -0.035247802734375, 0.33251953125, 1.6435546875, 0.274169921875, -0.58349609375, 0.54443359375, 0.465576171875, -0.16259765625, -1.048828125, -0.71435546875, -2.09375, 1.3955078125, 1.162109375, 1.0869140625, 1.8759765625, -1.9697265625, -0.028411865234375, 2.15234375, -0.18701171875, 1.8310546875, 0.716796875, 0.7138671875, 0.1326904296875, -1.255859375, -0.77294921875, 0.5517578125, -0.471923828125, 0.2491455078125, -1.9375, -0.515625, 0.36328125, 1.357421875, -0.35595703125, -1.322265625, 0.0027523040771484375, 1.595703125, 1.234375, 0.52880859375, -0.875, -0.3017578125, -1.3515625, 0.974609375, -0.75732421875, -2.7734375, -1.34375, 0.94580078125, 0.47265625, 1.021484375, -0.1654052734375, -1.3974609375, -1.0556640625, -0.313232421875, 0.0943603515625, -0.53515625, 0.1934814453125, -1.6484375, -2.615234375, 1.2099609375, 0.82470703125, -2.1171875, 0.76708984375, 1.087890625, 1.9404296875, 0.51123046875, -0.253173828125, 1.451171875, -0.12139892578125, 0.57568359375, 1.732421875, -0.2265625, -1.8095703125, 0.09100341796875, 1.2548828125, 0.7509765625, 1.8671875, -0.1754150390625, -0.1356201171875, 1.0634765625, 1.37890625, 2.400390625, 0.0758056640625, 0.6748046875, -1.302734375, 0.11663818359375, 0.499755859375, 1.6435546875, -0.464111328125, 0.990234375, -0.376953125, 0.7783203125, 0.90771484375, -0.984375, -0.4052734375, 0.2000732421875, 0.32958984375, -0.2410888671875, -1.5234375, -0.0826416015625, -1.130859375, 0.54931640625, 0.21044921875, 0.269775390625, 0.14306640625, 0.66015625, -1.166015625, -0.2081298828125, -0.65478515625, -0.70458984375, 0.69775390625, 1.294921875, 0.006343841552734375, -0.213623046875, -1.251953125, 1.3193359375, -0.183837890625, 1.046875, -0.50927734375, -0.428955078125, -1.6474609375, 0.10223388671875, -0.29833984375, -1.8857421875, 1.177734375, -0.058929443359375, 1.1494140625, 0.411865234375, 1.1259765625, 0.2021484375, 0.0838623046875, 0.90087890625, 0.323974609375, 0.208984375, -0.9501953125, -0.92626953125, 0.031463623046875, -1.7802734375, -1.3662109375, -0.5419921875, -0.310302734375, -0.69287109375, -0.12548828125, 0.89990234375, 0.865234375, 1.015625, 0.463134765625, -0.7294921875, -0.72802734375, -1.8544921875, -1.03515625, 0.2978515625, 1.1220703125, -0.051513671875, 0.320068359375, -0.66455078125, 0.95361328125, -0.377197265625, 2.09375, -0.7138671875, -0.276123046875, 0.488525390625, -0.537109375, 0.419677734375, 2.138671875, -0.99169921875, -0.399658203125, 2.177734375, -1.1474609375, -0.107421875, 0.6474609375, -0.450927734375, 0.8837890625, 1.2900390625, 0.75244140625, 0.056793212890625, -0.337646484375, 1.7294921875, -0.2158203125, 0.017608642578125, -2.154296875, -0.488525390625, 0.048492431640625, 0.31103515625, 0.48095703125, -1.6083984375, -0.0050506591796875, 1.609375, 0.484375, -1.349609375, 0.2203369140625, 0.11468505859375, 0.1171875, -1.7431640625, -0.70556640625, -0.7001953125, -2.048828125, 0.18994140625, 0.884765625, -1.099609375, -0.7666015625, -1.9150390625, -0.32470703125, 1.37109375, -0.68310546875, -2.162109375, -0.1365966796875, 0.55322265625, 0.4541015625, -0.354248046875, -1.4951171875, 0.4658203125, 0.451904296875, -0.59228515625, 0.82421875, 0.1134033203125, -1.19921875, 1.8642578125, 0.81201171875, -1.26953125, -1.025390625, -0.80224609375, -0.89794921875, 0.25830078125, -0.38525390625, 1.2724609375, 1.158203125, -0.91357421875, 0.57373046875, 1.041015625, -1.6435546875, 1.0478515625, 0.1300048828125, -0.005329132080078125, -0.060516357421875, -0.7412109375, -1.5888671875, -0.04937744140625, 0.259033203125, 0.352783203125, 0.59912109375, 1.1103515625, 0.64599609375, -0.576171875, -0.55810546875, 0.54931640625, -0.423828125, 1.412109375, -0.86083984375, 0.0038700103759765625, -0.12939453125, 0.7548828125, -0.78125, -1.4580078125, -0.462646484375, 0.377685546875, -1.4794921875, -0.180419921875, -1.7451171875, 0.124267578125, 2.25, -0.54150390625, -1.0009765625, 0.003086090087890625, -1.3232421875, -1.169921875, 0.7412109375, -0.56201171875, 0.64404296875, 0.484619140625, -0.48583984375, -0.10723876953125, -0.0234222412109375, 1.3251953125, 0.578125, -1.4765625, -0.55078125, 0.98779296875, 0.2147216796875, -0.859375, 0.1087646484375, -1.0634765625, -1.146484375, 0.32763671875, 0.58056640625, -0.77978515625, -0.1239013671875, 0.052093505859375, 0.64892578125, 0.45361328125, 1.052734375, -0.55322265625, -1.45703125, 0.83447265625, -1.7763671875, 0.728515625, 0.417724609375, -0.1673583984375, -0.5556640625, -0.08966064453125, -1.0, 0.99365234375, 1.41015625, -0.71435546875, 0.287841796875, 0.99462890625, -0.64306640625, -0.5322265625, 0.56396484375, 0.427490234375, -0.0350341796875, -0.4873046875, 0.55615234375, -0.5703125, -2.009765625, -0.56396484375, 0.81884765625, 0.52099609375, -1.9521484375, 0.1856689453125, 0.40380859375, 0.285400390625, 0.476806640625, -1.06640625, 0.311767578125, 1.8603515625, 0.30908203125, 0.45703125, 0.060516357421875, -0.95654296875, 0.05718994140625, -1.2626953125, 0.8544921875, -1.1416015625, 0.2333984375, -0.073974609375, -0.623046875, -0.826171875, 0.31640625, -0.78271484375, 0.78466796875, 0.720703125, 1.38671875, 0.20068359375, 1.1357421875, 0.2403564453125, 0.82861328125, 0.33984375, 0.06097412109375, -0.3671875, 0.0263824462890625, -0.34912109375, 1.16015625, 1.2998046875, 1.634765625, 0.1075439453125, -0.77099609375, 0.90283203125, -1.125, -0.8896484375, -1.1083984375, -0.93505859375, 2.4375, -0.027313232421875, 0.09954833984375, -1.189453125, 1.41015625, -0.37060546875, -0.67626953125, -0.1446533203125, -0.16015625, 0.99951171875, -0.2880859375, 0.189208984375, -1.6669921875, 1.3134765625, -0.7431640625, -0.70556640625, 1.3525390625, 0.054840087890625, -1.3388671875, -0.7880859375, 0.27099609375, -0.51904296875, 0.7998046875, 0.197998046875, -0.061798095703125, 0.380859375, 0.5126953125, -1.04296875, 0.6171875, -0.4208984375, -1.0908203125, 1.451171875, 0.92822265625, 0.09527587890625, -1.8291015625, 0.01409912109375, 0.5302734375, 1.1396484375, 1.783203125, -0.191162109375, 1.3798828125, 0.377685546875, 0.83203125, 0.759765625, -0.9052734375, 0.1619873046875, 0.1434326171875, -0.517578125, 0.2379150390625, 0.358154296875, 0.03668212890625, -0.28466796875, 0.34423828125, 0.501953125, 0.30615234375, -0.9150390625, 1.1572265625, 0.27783203125, 1.3193359375, -2.048828125, -0.0156402587890625, -1.353515625, 0.0301055908203125, 2.3515625, 0.84619140625, 1.205078125, 0.436767578125, -0.609375, -1.2548828125, -0.45556640625, -0.96533203125, 0.98095703125, 0.423828125, -0.61572265625, 0.67431640625, 1.630859375, 1.4189453125, 0.91552734375, 0.497802734375, 0.256591796875, 0.69384765625, 0.448486328125, 0.460205078125, 0.054656982421875, -0.373046875, 0.7294921875, -0.609375, 0.66455078125, -0.7294921875, -0.720703125, -0.1370849609375, 1.80078125, 0.49267578125, -0.21484375, -0.64208984375, -0.5712890625, -0.433349609375, 1.0517578125, 0.315185546875, 0.381591796875, 0.7412109375, 0.66650390625, 0.44091796875, -0.578125, 0.98681640625, -1.0126953125, -0.28955078125, -1.912109375, -1.45703125, 0.8291015625, 0.3740234375, -0.483642578125, 0.84912109375, -0.2626953125, -0.0172271728515625, 0.787109375, -0.228515625, 0.233642578125, 0.42919921875, -1.0849609375, 0.947265625, -0.46337890625, -0.192138671875, -0.5302734375, -1.0439453125, -0.55322265625, 0.1917724609375, 0.41259765625, -0.05364990234375, -1.2490234375, 0.3154296875, 0.1011962890625, 0.72802734375, -1.7080078125, -0.9599609375, -0.3544921875, -0.305419921875, -1.71484375, 0.467529296875, -1.576171875, 0.47412109375, -1.6552734375, 0.484130859375, 0.67041015625, -1.0556640625, -0.55517578125, -0.45556640625, -0.415771484375, -1.271484375, 0.00412750244140625, -1.4306640625, 0.7080078125, -1.3359375, -1.0, 1.4248046875, -0.0141754150390625, -0.64599609375, 0.48876953125, -1.3818359375, -0.900390625, -0.7333984375, 0.11962890625, 1.494140625, 1.2353515625, 0.91552734375, -0.10662841796875, -1.3740234375, 0.69384765625, -0.0721435546875, 0.20849609375, -0.375244140625, -0.54833984375, -0.62548828125, 0.9794921875, -0.3046875, -1.2392578125, 0.2470703125, -0.8095703125, 0.5166015625, 0.72900390625, 2.263671875, -0.59716796875, -0.8212890625, 0.41162109375, -0.448974609375, -0.619140625, -0.1890869140625, 0.66552734375, 1.0947265625, 0.583984375, 1.3330078125, 0.982421875, -1.75, -0.3759765625, -0.73193359375, -0.58447265625, -1.0048828125, -0.462646484375, 0.071044921875, -0.417236328125, -0.8388671875, 0.05804443359375, 0.60400390625, 0.09857177734375, 1.32421875, 0.865234375, 1.5478515625, 1.7412109375, -0.57666015625, -0.062042236328125, 0.2237548828125, 0.10406494140625, -1.154296875, 1.6875, 0.1549072265625, -0.7158203125, 0.2098388671875, 0.19970703125, 1.3203125, -0.7734375, -1.7998046875, -0.859375, -0.08648681640625, 1.2861328125, 0.5146484375, -0.6865234375, -0.343994140625, -0.998046875, -0.3046875, -0.48095703125, 0.17822265625, -0.47705078125, 0.01898193359375, -0.397216796875, 1.2919921875, 0.004230499267578125, 2.46484375, -0.255615234375, 0.09136962890625, -1.4306640625, -0.485107421875, 0.08160400390625, -1.509765625, -0.387451171875, 0.8095703125, -0.36865234375, 0.07659912109375, -1.6240234375, -2.61328125, 0.84375, -0.1810302734375, 0.2296142578125, -0.3564453125, -0.1197509765625, -0.341796875, 1.138671875, -1.044921875, 2.6015625, -0.3056640625, -0.54931640625, 0.470458984375, 1.0634765625, 0.0106201171875, 0.1934814453125, 0.71142578125, 0.044158935546875, 0.196533203125, -0.2666015625, -2.451171875, 0.55517578125, 0.68603515625, 0.148681640625, -0.474853515625, -0.483642578125, 1.6181640625, 2.6171875, 1.16796875, -0.035125732421875, 1.19921875, -1.1943359375, -1.142578125, 0.433837890625, 1.2275390625, 1.9755859375, -0.463134765625, -1.5986328125, -1.2490234375, -0.04205322265625, -0.473388671875, 0.5546875, 0.5986328125, -0.17626953125, -1.7685546875, 0.193115234375, -0.0176849365234375, -0.81201171875, -1.0732421875, 0.280517578125, 0.45458984375, 0.60107421875, -0.157470703125, -0.032318115234375, -0.71923828125, -2.060546875, 0.045501708984375, -0.76220703125, -0.00524139404296875, -0.434814453125, -2.158203125, -0.92529296875, 0.67724609375, 1.2783203125, 0.432373046875, 0.53369140625, -0.23876953125, -0.128173828125, -1.7353515625, 2.40625, -0.1632080078125, -1.4560546875, -0.2401123046875, 1.9619140625, 0.3828125, 1.51171875, 1.6171875, 1.482421875, 0.5078125, -0.1859130859375, -0.386962890625, -1.5361328125, 0.06396484375, -0.37744140625, -0.1527099609375, 1.72265625, -1.0673828125, 0.2431640625, -0.76513671875, 0.004444122314453125, -0.4140625, 0.1627197265625, 1.1005859375, -0.69580078125, -0.96337890625, -1.5166015625, 1.3642578125, 0.192138671875, -2.435546875, -0.64306640625, 0.97314453125, -0.459716796875, -1.1044921875, -1.0439453125, 0.456787109375, -0.7216796875, -0.237548828125, 0.197021484375, 1.26171875, 0.8076171875, 0.56640625, -0.314208984375, 0.36279296875, 0.1336669921875, -0.7265625, -1.1025390625, -0.77880859375, -0.666015625, -0.382080078125, 1.083984375, 1.2451171875, -1.3662109375, 1.4306640625, -0.303955078125, 0.1947021484375, -0.451416015625, 1.416015625, 0.401123046875, -1.0126953125, 0.03515625, 1.1796875, -0.82470703125, -0.007236480712890625, -0.70556640625, 0.77685546875, 0.231689453125, -1.521484375, -1.63671875, -1.599609375, -0.9658203125, 3.224609375, 0.91259765625, -0.84326171875, 1.96484375, -1.01953125, -0.296875, 0.82373046875, -1.6904296875, 0.406005859375, 1.7236328125, -0.0751953125, -0.60546875, -0.82666015625, -0.57177734375, 1.388671875, 0.5849609375, -1.046875, -0.278564453125, 0.8193359375, 0.34912109375, -0.642578125, -0.623046875, 1.4794921875, 0.9501953125, 0.076904296875, -0.375244140625, 0.07916259765625, -0.23828125, -1.615234375, 0.04180908203125, 1.6865234375, 1.509765625, 2.25, 0.00905609130859375, -0.83740234375, -1.8125, 0.89208984375, -0.15576171875, -0.2607421875, 0.85986328125, -0.640625, -0.2401123046875, 0.79736328125, -1.1904296875, 1.79296875, -1.4951171875, -0.2401123046875, -1.0693359375, -1.31640625, 2.228515625, -1.337890625, 0.828125, -1.4208984375, -0.716796875, -0.8203125, 2.326171875, -1.1005859375, -0.5615234375, -1.2783203125, 0.294677734375, 0.362060546875, 0.357666015625, 0.1888427734375, 0.8828125, 0.2474365234375, -0.84619140625, -0.908203125, 0.78369140625, -3.3984375, -0.7041015625, 0.259033203125, 0.401611328125, 0.363525390625, 0.4208984375, 1.3759765625, 0.440185546875, 1.2021484375, 0.1790771484375, 0.14453125, -1.2119140625, 0.50732421875, -0.1475830078125, -0.2685546875, 0.16796875, 0.64111328125, 0.953125, 1.087890625, -0.7802734375, -0.2325439453125, -0.1199951171875, 0.06927490234375, -0.46044921875, -0.6826171875, -1.6484375, 0.24658203125, -0.46142578125, -1.1494140625, 1.44921875, 2.2734375, 0.231201171875, -1.048828125, -0.517578125, -1.4697265625, -0.65087890625, 1.1064453125, -1.955078125, -0.1572265625, 1.7392578125, -1.5654296875, -0.291748046875, 0.7646484375, -1.181640625, -1.0244140625, 0.6494140625, 0.56591796875, -0.244873046875, -0.6484375, -0.599609375, -0.162841796875, -0.5439453125, -1.4560546875, -1.32421875, 0.61328125, 0.270751953125, 2.015625, -0.53271484375, -0.0209503173828125, -1.3251953125, 0.0726318359375, 0.810546875, 1.75390625, -0.6044921875, -0.288818359375, -0.044097900390625, 0.6162109375, -0.277099609375, 0.35009765625, -1.6376953125, -0.8388671875, 0.105224609375, -0.7919921875, -0.94189453125, 0.0256195068359375, -0.453125, 0.06707763671875, -0.9833984375, -1.26953125, 0.15771484375, 1.5087890625, -1.2587890625, 1.833984375, -0.90673828125, -0.05816650390625, 2.29296875, 0.52783203125, -0.08984375, 0.204833984375, -0.78076171875, -1.544921875, -1.09765625, -1.1728515625, -0.2296142578125, 0.73779296875, 0.5791015625, -1.4287109375, 1.1787109375, 0.2431640625, 0.44677734375, 0.755859375, -0.2783203125, 0.99609375, 1.310546875, 0.43212890625, -1.0673828125, -0.1754150390625, 1.224609375, 0.921875, -0.80029296875, 0.332275390625, -1.3115234375, 0.9248046875, -0.1488037109375, -0.7470703125, 1.3369140625, -0.70947265625, 1.2734375, -1.7626953125, 0.5205078125, -0.6416015625, 0.226318359375, 0.142822265625, -1.017578125, -0.90673828125, 2.224609375, -0.482666015625, 0.2142333984375, 0.796875, 1.6748046875, 0.08544921875, 0.436767578125, -0.034393310546875, 0.386962890625, -0.46435546875, 0.2744140625, -1.966796875, -0.74755859375, -0.82666015625, 0.61865234375, 0.5185546875, 1.083984375, -0.359375, -1.01171875, -0.70263671875, -0.740234375, 1.689453125, -1.18359375, 1.6708984375, 1.220703125, -0.0877685546875, -0.853515625, 0.0101776123046875, -1.0283203125, 0.1575927734375, 1.6259765625, 1.134765625, -0.974609375, 0.1859130859375, 1.494140625, 0.8681640625, 0.2890625, 0.78076171875, 0.83447265625, -0.059356689453125, -0.330322265625, 0.56494140625, -0.8642578125, -0.845703125, -0.970703125, 0.70166015625, 0.11322021484375, -0.7724609375, 1.7529296875, 0.061614990234375, -0.80908203125, -1.677734375, -0.51318359375, -0.2548828125, -0.260986328125, -0.16748046875, 0.80029296875, 1.1064453125, 0.36572265625, 1.2138671875, -0.067626953125, 0.470947265625, 0.90771484375, 1.119140625, 0.96630859375, -1.8310546875, 0.615234375, -1.1337890625, -0.219482421875, -0.494140625, -1.009765625, -1.421875, 0.39892578125, -0.54638671875, 0.392578125, -0.7880859375, -1.2275390625, 0.10577392578125, 1.361328125, -0.76318359375, -0.12890625, 1.0888671875, 0.94091796875, 1.02734375, 1.37109375, 0.249267578125, -0.78955078125, 0.2626953125, 0.80322265625, -0.7060546875, -0.01253509521484375, -0.48974609375, 1.419921875, -0.6357421875, 0.69873046875, 0.57470703125, -0.24951171875, 0.1846923828125, 0.49609375, 0.595703125, 1.302734375, -0.5703125, 1.029296875, 0.3056640625, 0.4482421875, 1.341796875, 0.5166015625, -1.3095703125, -0.2890625, 0.56103515625, -0.45166015625, 0.1484375, -0.6630859375, -0.5888671875, 0.7802734375, 0.86767578125, 1.791015625, 0.5751953125, -0.05908203125, -0.23486328125, -1.0712890625, 0.955078125, -0.251220703125, 0.23388671875, -1.6376953125, 0.247802734375, -1.7900390625, 0.1038818359375, 1.94140625, 1.2041015625, 1.0986328125, 0.51806640625, 0.0086517333984375, 1.1171875, 0.716796875, 0.04986572265625, -1.279296875, -0.2044677734375, -1.650390625, 1.3603515625, 0.595703125, -0.244873046875, -0.76171875, 0.62451171875, 0.32568359375, -0.35400390625, -0.88037109375, 0.94482421875, -0.024017333984375, -0.41748046875, -1.7744140625, -0.100830078125, 0.1177978515625, -0.277099609375, 0.6640625, -0.328125, -0.990234375, -0.05126953125, 0.043975830078125, 0.77734375, 0.00547027587890625, 1.1474609375, -1.3623046875, -0.287353515625, 0.86865234375, -1.4130859375, -0.93505859375, -1.0146484375, -0.10601806640625, -0.453125, 2.25, -0.12548828125, 0.634765625, 0.1373291015625, -1.65234375, -1.0419921875, -0.347412109375, 0.016021728515625, -0.039215087890625, -0.53076171875, -1.6982421875, -0.227294921875, 0.4736328125, 0.44775390625, 1.1689453125, 0.6806640625, 1.77734375, -0.298583984375, -0.361328125, 1.650390625, 0.389892578125, -0.7861328125, -1.296875, 0.53466796875, -0.278076171875, -0.11236572265625, -0.89306640625, -0.25537109375, 0.6572265625, -1.828125, -1.517578125, -0.80419921875, -0.295654296875, 0.2410888671875, 0.5859375, -0.442626953125, -0.7216796875, 1.8564453125, -1.224609375, 0.26171875, 0.7255859375, -0.97412109375, -1.6728515625, 1.7705078125, -0.53271484375, -1.2333984375, -1.080078125, -0.054534912109375, -0.95654296875, 2.375, 0.1600341796875, -0.8642578125, 0.69580078125, -0.830078125, 1.1904296875, 0.302490234375, -1.279296875, 0.66650390625, -1.8564453125, -0.4208984375, 1.1240234375, -0.40185546875, -1.5771484375, 0.892578125, 1.017578125, 0.12261962890625, 1.1572265625, -0.248046875, 0.74462890625, -0.264892578125, 1.037109375, -0.291748046875, -0.482421875, -0.1590576171875, -1.12109375, 1.3017578125, 0.515625, -0.490966796875, 0.405029296875, -1.5068359375, -1.3056640625, -2.2578125, -1.1943359375, -1.0673828125, 0.1302490234375, -0.229736328125, -0.1451416015625, -0.556640625, -0.2244873046875, -0.67138671875, -1.5771484375, 2.7265625, 0.166259765625, -0.8583984375, 1.7587890625, -1.8681640625, -0.806640625, 1.123046875, -0.1480712890625, 0.24951171875, -1.34375, -0.417236328125, 0.419677734375, 0.425048828125, 0.9970703125, -0.26904296875, 0.410888671875, 0.496826171875, 0.94921875, 0.52880859375, -0.767578125, 0.297119140625, 0.63330078125, -1.7158203125, -0.6123046875, 0.837890625, 1.333984375, -0.385986328125, 1.1533203125, -1.7060546875, -0.41015625, -0.43994140625, 1.2724609375, 0.880859375, -1.0693359375, -0.498291015625, -1.1982421875, 0.07958984375, 0.095458984375, -1.8642578125, -0.438232421875, -1.7138671875, -0.26220703125, -0.0257568359375, -1.146484375, -1.0029296875, -0.245361328125, 0.35302734375, 0.68115234375, 0.9306640625, 0.90869140625, -0.755859375, -0.34716796875, 1.14453125, -1.2216796875, -0.6953125, -0.07373046875, -1.015625, -0.62646484375, -0.1304931640625, -2.3125, -0.311279296875, 0.35009765625, 1.3994140625, -1.4638671875, -0.3896484375, -0.1405029296875, 1.6376953125, -0.9052734375, 0.399169921875, 0.316650390625, -0.38671875, -0.4599609375, -1.8203125, -1.232421875, -0.07080078125, -0.240966796875, -0.654296875, 0.2147216796875, -0.52197265625, -0.55029296875, 0.5693359375, -1.732421875, 0.1309814453125, 0.5419921875, -0.77587890625, 2.0078125, 0.408447265625, 0.02716064453125, 0.09490966796875, 0.305908203125, -0.49755859375, -0.66796875, -0.1998291015625, -0.262451171875, -0.57958984375, -0.88818359375, -1.5869140625, -0.2286376953125, 0.0855712890625, 0.69677734375, 0.71337890625, -0.1572265625, -0.74658203125, -0.6572265625, -0.1671142578125, 1.4453125, 0.0293426513671875, -1.6083984375, 0.904296875, 0.44677734375, -1.2177734375, 0.0185699462890625, 0.4169921875, -1.4501953125, 0.270751953125, 0.2017822265625, -0.2978515625, 0.59228515625, -0.308837890625, -1.03125, -0.10491943359375, 0.8916015625, -0.85205078125, -0.85107421875, -0.92333984375, -0.576171875, -0.2410888671875, 0.68798828125, -2.38671875, 0.88525390625, -0.884765625, -0.079833984375, 0.388427734375, -2.177734375, 0.748046875, 0.14306640625, -0.50439453125, 2.517578125, 1.6103515625, -0.35986328125, 0.00872039794921875, -0.0114288330078125, 0.408935546875, 0.352783203125, 0.275146484375, -1.8544921875, -2.109375, -0.359375, -1.0771484375, 0.106689453125, -0.318603515625, -0.371337890625, -0.61767578125, 0.053558349609375, 0.284423828125, 0.779296875, -0.173095703125, -2.166015625, -1.578125, -0.732421875, -0.080322265625, 0.2318115234375, -0.01287841796875, 1.1591796875, -0.53662109375, -1.16796875, 0.37646484375, -1.2392578125, 0.9267578125, 0.234619140625, -0.67041015625, -0.356689453125, 0.59619140625, 0.369873046875, -1.6337890625, -0.69091796875, -0.30126953125, -0.03216552734375, 1.572265625, 0.8564453125, -1.607421875, 0.46044921875, -0.51806640625, -0.1590576171875, 0.5087890625, -1.2119140625, -0.77587890625, -0.99609375, -0.27685546875, 1.3251953125, 1.4462890625, 0.293212890625, 0.9365234375, 0.83203125, 1.154296875, 0.1607666015625, 0.04913330078125, -1.6953125, -0.5234375, -0.2052001953125, -0.0438232421875, 0.94287109375, 0.541015625, -0.318115234375, -1.2294921875, -0.426025390625, 0.80419921875, -0.697265625, -0.162353515625, -0.88427734375, 0.447265625, 2.14453125, -0.466552734375, 0.5009765625, 0.54248046875, 1.2109375, 2.541015625, 1.185546875, 1.10546875, 0.5517578125, -0.6123046875, 0.0804443359375, -0.049713134765625, -2.021484375, -0.72607421875, 0.38134765625, -0.9560546875, 1.640625, 1.205078125, -2.1796875, 1.3173828125, 0.7548828125, -1.533203125, -0.2435302734375, -0.0014028549194335938, -1.240234375, 2.75390625, 1.4638671875, -0.427734375, 0.5498046875, -0.89404296875, 1.1884765625, -1.7216796875, -0.62548828125, -0.08856201171875, -0.6669921875, 0.263427734375, 0.583984375, -1.7578125, -0.89404296875, -0.0992431640625, 1.203125, 1.5390625, 0.449462890625, 1.2900390625, -0.8212890625, -0.023590087890625, 0.5712890625, 1.0966796875, 0.204833984375, 0.2294921875, 0.28173828125, 0.153076171875, 0.35498046875, 1.9970703125, 0.96875, -0.486572265625, -0.377197265625, 0.720703125, 0.07354736328125, 0.274658203125, 1.0458984375, 1.70703125, 1.3857421875, -0.473876953125, 0.732421875, 0.1612548828125, 1.248046875, -1.1279296875, -0.11651611328125, 0.7119140625, 0.384765625, 0.7158203125, -0.28759765625, 0.35498046875, -0.493896484375, -0.2191162109375, 1.8662109375, -1.4384765625, -1.0966796875, -0.984375, -0.1358642578125, 1.349609375, -1.0732421875, -1.373046875, 0.86865234375, 0.255126953125, -0.76611328125, -0.018218994140625, 0.97900390625, 0.8916015625, -0.9775390625, 0.269775390625, -0.4599609375, -1.42578125, -0.689453125, 0.372314453125, -0.153076171875, 0.156005859375, 0.4091796875, 0.237060546875, -0.1556396484375, -0.1612548828125, -0.0494384765625, -1.306640625, 0.64306640625, -0.6123046875, -1.0234375, 1.27734375, -0.167236328125, -0.8681640625, -0.48828125, 0.67724609375, -0.6337890625, -0.63916015625, -0.95751953125, -1.3134765625, -0.3623046875, -0.845703125, -1.4775390625, 0.81787109375, -1.072265625, -0.56787109375, -1.5966796875, -0.66015625, 0.5732421875, -0.6162109375, 0.26904296875, 1.6640625, -0.99267578125, -1.4033203125, 0.382568359375, -0.5556640625, 0.619140625, -1.8251953125, -2.267578125, -1.2880859375, -1.2734375, -0.331787109375, -0.2607421875, -0.07781982421875, -0.110107421875, -0.3447265625, -1.556640625, -0.314453125, -0.427978515625, 1.08203125, -0.77587890625, 1.126953125, -0.33056640625, 1.1611328125, -0.2276611328125, -0.8974609375, -0.7509765625, -0.943359375, -0.3974609375, -0.4921875, -0.474609375, -0.1456298828125, 1.369140625, 0.67138671875, -1.30859375, 0.94775390625, -0.84033203125, -0.8232421875, -0.90478515625, 0.8330078125, 1.2197265625, 0.728515625, -0.0767822265625, 0.473388671875, -0.60888671875, -0.374755859375, -0.052764892578125, -0.7158203125, -0.64453125, -0.61181640625, 0.005107879638671875, 0.7607421875, -0.94775390625, 1.060546875, -0.68115234375, -2.048828125, -0.158203125, 0.6513671875, -0.125244140625, -1.08984375, 0.72509765625, -2.298828125, 0.9765625, 0.93505859375, 0.5009765625, 0.175048828125, 1.466796875, -0.2783203125, 0.8515625, -0.13427734375, -1.0517578125, -0.0226287841796875, 1.5712890625, -1.0732421875, -0.202880859375, 0.135986328125, -1.1923828125, 1.599609375, -1.9677734375, -0.83349609375, -1.0068359375, 2.3203125, -0.68798828125, -1.0107421875, 0.6796875, 1.115234375, 0.223388671875, -0.146728515625, 0.3720703125, -0.1856689453125, 1.123046875, 0.77294921875, -0.47802734375, -0.91357421875, 0.08050537109375, 0.1265869140625, 0.70458984375, 0.57763671875, -1.6015625, 1.6494140625, 0.1575927734375, 0.53759765625, -0.159912109375, 1.544921875, -3.0234375, -1.046875, -0.642578125, -0.1251220703125, 0.12042236328125, -0.74267578125, -0.0290374755859375, 0.64794921875, 0.3623046875, 0.9208984375, -0.68359375, 1.919921875, -0.27587890625, -0.2105712890625, 0.6533203125, -0.2310791015625, -1.0712890625, -0.69140625, 0.086669921875, 0.0931396484375, 1.2685546875, -0.55419921875, 0.57177734375, 1.8203125, -0.2379150390625, -1.576171875, 1.025390625, -2.478515625, 0.054473876953125, 0.74853515625, 0.484375, -0.24609375, -1.1494140625, -0.0765380859375, -1.2607421875, -0.0290679931640625, 0.91796875, -0.3662109375, 0.31884765625, 0.65673828125, 0.198486328125, 0.6806640625, 1.5, 2.08203125, -1.798828125, 0.1663818359375, 0.8505859375, 0.88671875, -0.7890625, -0.96240234375, -0.14306640625, -1.166015625, -0.272705078125, -0.7919921875, 1.01953125, -0.300537109375, -1.201171875, -1.6044921875, -1.5908203125, 1.40625, 0.814453125, -0.83544921875, -0.88134765625, 1.490234375, 0.53955078125, -1.5693359375, -0.70068359375, 0.5439453125, 3.21484375, 0.418212890625, 2.08203125, 0.1759033203125, 1.86328125, -0.490234375, -0.8349609375, -0.46533203125, -1.068359375, 0.80078125, 1.3134765625, 0.29052734375, -0.662109375, 0.1968994140625, -1.583984375, 0.59423828125, 1.2734375, -0.2178955078125, 1.5625, 0.0701904296875, -1.333984375, -0.086669921875, -0.53564453125, -0.8466796875, 0.68896484375, 2.783203125, 0.293212890625, -1.3427734375, 1.2939453125, 1.2109375, -0.019378662109375, -0.3154296875, -1.0390625, -0.99658203125, 0.9404296875, 1.87890625, 0.53466796875, -0.42138671875, -0.49365234375, -0.281982421875, -1.46484375, 0.181396484375, 1.6015625, -1.689453125, 0.6328125, 1.09765625, 0.1981201171875, 1.7177734375, -1.8564453125, 1.1728515625, -0.62060546875, 0.15380859375, 0.62939453125, 0.96240234375, -0.06573486328125, 0.447265625, -0.364501953125, -0.9365234375, -1.03515625, 0.46826171875, 0.190673828125, 0.94482421875, 2.60546875, 1.2255859375, -0.5166015625, 0.88037109375, 0.59423828125, 0.84130859375, 0.368408203125, -1.0849609375, 1.0263671875, 0.285400390625, -0.705078125, 0.8740234375, 0.5712890625, -1.1259765625, -0.560546875, -1.9365234375, -0.61328125, -0.41162109375, 0.481689453125, -1.0234375, -0.62744140625, -0.09783935546875, -0.468994140625, -0.44775390625, 1.70703125, 0.252685546875, 0.131103515625, 0.9599609375, -0.89111328125, 0.7705078125, -0.6650390625, -0.8740234375, 0.87890625, 0.029266357421875, 0.0161895751953125, -1.3525390625, 1.42578125, 0.5791015625, 0.281494140625, 1.265625, -2.095703125, 0.859375, 1.7529296875, 0.285888671875, -0.62548828125, -2.5078125, -0.212158203125, 1.435546875, -0.1953125, -0.271240234375, 0.393798828125, -0.2008056640625, 0.232177734375, -0.08709716796875, -0.2415771484375, 0.57373046875, 1.0322265625, 1.546875, -1.5732421875, 0.88671875, -0.33740234375, 2.0546875, 0.308837890625, 0.4013671875, 1.5810546875, -0.716796875, -1.2548828125, -2.66796875, -1.5908203125, 1.9580078125, -0.6357421875, -0.094970703125, -1.525390625, -0.1695556640625, 0.43310546875, -1.912109375, -0.391845703125, 0.47265625, 0.322265625, -0.546875, -0.74462890625, 0.024566650390625, -0.08642578125, 1.3564453125, -1.2119140625, -0.39501953125, -0.270263671875, -1.015625, -1.4638671875, 0.1341552734375, 0.319091796875, 0.2022705078125, 0.045562744140625, 0.12890625, 0.68701171875, -0.7109375, -0.42822265625, -0.226318359375, -1.673828125, -1.6005859375, 0.8212890625, -0.0181884765625, -0.44775390625, 0.72607421875, 1.0546875, 2.041015625, 0.04443359375, 0.20703125, -0.619140625, -1.091796875, -1.2607421875, 1.0322265625, -1.5556640625, 0.18359375, -0.45947265625, 0.2296142578125, -0.1744384765625, 0.83837890625, 0.1326904296875, -0.03350830078125, -0.81005859375, -0.4072265625, 2.462890625, 0.2447509765625, 2.576171875, 1.677734375, 1.216796875, 0.841796875, 0.0872802734375, 0.404052734375, 0.393798828125, 0.9951171875, -0.6708984375, -0.9609375, -0.71435546875, 0.326416015625, 1.2685546875, 0.033599853515625, -1.8515625, -0.265380859375, 0.2626953125, 0.59423828125, 0.346923828125, -0.49365234375, -2.84375, -1.603515625, -0.06854248046875, -0.73681640625, -1.3037109375, -2.33203125, -0.51416015625, 1.8095703125, -0.8154296875, 0.9921875, 1.1640625, 0.1513671875, -0.8193359375, -1.0908203125, 0.30029296875, 1.48046875, -2.2890625, 0.8876953125, -0.443603515625, -1.0498046875, -1.14453125, -0.5830078125, 0.67919921875, 0.67236328125, 0.274658203125, -1.05078125, -0.94140625, 1.6015625, 0.28857421875, 0.40673828125, -1.14453125, -0.29931640625, 0.9365234375, -0.9580078125, 1.2470703125, 0.93896484375, -0.248291015625, -0.990234375, 0.266845703125, -0.85302734375, -0.7666015625, -0.440673828125, 1.890625, -0.68115234375, 0.646484375, -1.17578125, 1.033203125, -1.126953125, -1.189453125, -0.0709228515625, 0.423828125, 1.0244140625, -0.6943359375, -0.01515960693359375, 1.4228515625, -2.1875, -0.366943359375, 1.5498046875, -0.0858154296875, 0.127197265625, 0.291748046875, -0.375732421875, -0.21484375, 0.10711669921875, -0.35009765625, -0.96484375, 1.2958984375, -0.703125, 0.2210693359375, 1.271484375, 0.451171875, 1.6689453125, -0.00913238525390625, -0.1407470703125, 1.244140625, 1.7646484375, 0.9423828125, -0.476806640625, 1.697265625, -3.314453125, -0.498291015625, 0.888671875, 0.42919921875, 0.2325439453125, 0.353759765625, -1.3212890625, -1.2197265625, -0.1878662109375, -0.67431640625, -0.78125, -0.111083984375, -0.67431640625, -1.0244140625, -0.71923828125, 1.09765625, -1.7109375, -0.8095703125, -0.8310546875, -1.29296875, 0.67724609375, 0.52001953125, -0.994140625, 0.04693603515625, -1.62890625, 1.2958984375, -0.05450439453125, -1.6796875, -0.8310546875, 0.89501953125, -0.70947265625, -0.2088623046875, -0.002895355224609375, -0.6123046875, 0.80810546875, 1.2333984375, -1.224609375, 0.489013671875, -0.43994140625, -0.303466796875, -0.54248046875, 0.1036376953125, 0.79931640625, 1.0791015625, 0.304443359375, -1.423828125, 0.476318359375, 0.8525390625, -0.79296875, 0.423583984375, -0.57958984375, 0.8486328125, -1.869140625, -0.338134765625, 0.66015625, 0.470703125, -0.023712158203125, -0.3486328125, -0.1597900390625, 0.599609375, 1.123046875, 0.62109375, -0.845703125, -0.0029239654541015625, 0.30517578125, 0.97314453125, 1.4892578125, -0.98095703125, 0.482177734375, -1.03515625, 0.779296875, -1.5556640625, 1.353515625, 1.0078125, 0.654296875, -0.857421875, -1.33984375, -1.2509765625, 0.1920166015625, 0.8935546875, -0.0152130126953125, 0.5546875, 0.8984375, 0.1104736328125, 0.140869140625, -0.43701171875, -0.24609375, -1.69921875, 0.0499267578125, 1.6728515625, -0.86962890625, -0.64697265625, 0.0791015625, -0.638671875, -1.072265625, -0.279052734375, -0.2509765625, 0.76171875, -1.3994140625, 0.9296875, -0.82421875, -0.97802734375, -0.33935546875, -0.10540771484375, 0.1494140625, -0.8232421875, 0.131103515625, 1.5771484375, 0.2457275390625, 0.013031005859375, 0.2177734375, 0.035797119140625, -0.276123046875, 0.88671875, -1.1376953125, -1.5439453125, 0.68359375, 1.337890625, 1.3828125, -1.0751953125, 0.06103515625, 0.09991455078125, -0.255615234375, 0.37548828125, 1.0224609375, 1.0380859375, -0.66552734375, 1.4013671875, 0.0033473968505859375, 0.1300048828125, 0.58203125, 0.54736328125, 0.436279296875, -0.498291015625, -1.642578125, -1.2666015625, -0.274658203125, 0.11669921875, -0.6162109375, 1.71484375, 1.533203125, 0.0982666015625, -0.177490234375, 1.73828125, 0.05474853515625, -0.89501953125, -1.4580078125, -0.85205078125, 0.70556640625, -1.7412109375, -1.087890625, -0.810546875, 0.75439453125, -0.37646484375, 1.2744140625, -0.61767578125, -0.19287109375, 1.1845703125, -0.005550384521484375, 0.05035400390625, -1.2705078125, -0.386474609375, -0.89990234375, -2.001953125, 0.2841796875, 0.93701171875, 0.58349609375, -1.4697265625, -1.4775390625, -0.68212890625, 0.7841796875, -0.09814453125, -0.70849609375, 0.6748046875, 0.304931640625, -0.5908203125, 2.455078125, 0.59423828125, -0.3076171875, -0.1278076171875, 1.849609375, -0.39013671875, 0.0225982666015625, 0.451416015625, 0.54736328125, -0.72998046875, -1.8828125, -1.3203125, -0.458984375, 1.4189453125, -0.66748046875, -0.03369140625, -0.93896484375, -0.9619140625, 2.6328125, 0.5458984375, 1.271484375, 0.51953125, 1.5673828125, 0.23876953125, 0.8095703125, 1.48828125, -0.505859375, -0.87841796875, 0.054931640625, 0.139892578125, -0.01233673095703125, -0.7275390625, -0.40087890625, 0.0823974609375, -2.08203125, 0.20751953125, -0.400146484375, -1.05859375, -1.68359375, 0.69873046875, 0.619140625, 0.57373046875, -0.654296875, -0.775390625, -0.90576171875, 0.888671875, 2.517578125, 0.427734375, 0.1602783203125, -0.76220703125, -1.5166015625, -0.669921875, 3.6015625, -0.449951171875, 0.2060546875, 0.30859375, 0.406005859375, 0.191162109375, 0.06658935546875, 0.197509765625, 0.4326171875, -0.98779296875, 0.09716796875, 1.484375, 0.40625, 1.13671875, -2.443359375, 0.047607421875, -1.96484375, -0.18798828125, -0.98974609375, -0.453125, 1.513671875, 0.029083251953125, 0.2183837890625, 0.359619140625, 0.0875244140625, 1.119140625, -2.19921875, 0.0264739990234375, -0.2003173828125, 0.982421875, -0.57958984375, 0.049346923828125, -1.43359375, 0.1517333984375, 1.73828125, -0.05926513671875, -0.2264404296875, -0.84130859375, 0.8544921875, 2.396484375, -0.9384765625, -0.7880859375, -0.451171875, -0.242431640625, -0.74169921875, -1.09375, -0.5322265625, 0.12646484375, 0.96435546875, -1.359375, -0.6689453125, -1.1044921875, -0.8984375, -1.025390625, -0.93505859375, 0.08026123046875, -0.83984375, 0.326416015625, -0.7880859375, -2.271484375, -0.482421875, -0.84912109375, 1.5029296875, 0.0260772705078125, -0.95849609375, -1.966796875, -0.263671875, -1.5205078125, -1.28515625, 1.177734375, -2.5546875, -0.5498046875, -0.7529296875, -0.403076171875, 0.04486083984375, -0.1834716796875, -1.4296875, -0.79638671875, -0.19873046875, 0.68896484375, -1.275390625, 0.2371826171875, 0.548828125, 0.712890625, -1.0625, -1.1787109375, 0.449462890625, 0.68408203125, -1.9521484375, -0.363037109375, -0.34814453125, -1.6279296875, -0.935546875, 1.2265625, -1.5400390625, -0.51416015625, -0.4931640625, 0.75537109375, 1.599609375, 0.69140625, -1.595703125, -1.9501953125, -0.457763671875, 0.384033203125, 1.341796875, 0.1685791015625, -0.435302734375, 0.20556640625, -1.0205078125, -0.1617431640625, -1.490234375, 0.7099609375, -0.755859375, 0.7646484375, -0.576171875, -2.28125, 0.463623046875, 0.75537109375, 1.0947265625, -1.064453125, -0.66650390625, -2.966796875, -1.5078125, -1.5888671875, 0.400390625, -0.95947265625, -0.264892578125, 0.60107421875, -1.619140625, -0.9052734375, -0.9580078125, -0.294921875, -2.15625, 0.78955078125, -1.1806640625, 0.64111328125, 0.233642578125, 0.33203125, -0.5537109375, -0.416015625, 0.438720703125, 0.9658203125, -0.41796875, -0.8369140625, 0.145751953125, -0.70166015625, 0.381103515625, -0.046661376953125, -1.1669921875, -2.353515625, -0.194580078125, 0.70458984375, 0.424560546875, -0.6982421875, 0.90625, -1.486328125, -0.943359375, 0.03668212890625, 0.998046875, 0.0321044921875, -2.076171875, -0.92919921875, 0.283447265625, 0.791015625, -0.97802734375, 3.232421875, -0.0853271484375, -0.274658203125, 0.69140625, 1.296875, 0.2200927734375, 0.24365234375, 0.455322265625, -0.490478515625, -0.52783203125, -0.0419921875, -1.1611328125, 0.055633544921875, -0.60546875, -0.0189971923828125, -0.9404296875, -0.177734375, -0.82275390625, 0.61572265625, -0.7041015625, -0.313232421875, 1.58984375, -0.0022125244140625, 0.114501953125, 0.88525390625, -0.0751953125, 1.2431640625, -1.0146484375, 0.369873046875, -1.1650390625, -0.58154296875, 0.401123046875, 1.1025390625, -2.04296875, 0.8291015625, -0.59619140625, -0.54541015625, 0.413818359375, 0.76220703125, -0.62890625, 0.53955078125, 0.327392578125, -2.05078125, 1.9443359375, -0.08953857421875, -1.4833984375, 1.345703125, -2.615234375, -0.57275390625, 0.306884765625, 0.6826171875, -1.7646484375, -0.1156005859375, 0.0271453857421875, -0.38427734375, 0.97802734375, 0.378173828125, 0.833984375, -1.7646484375, -0.11279296875, 1.28515625, 1.1953125, 0.513671875, -0.79833984375, -0.17529296875, 1.67578125, -1.2451171875, -1.517578125, -0.90625, 0.541015625, 0.2115478515625, -0.6142578125, -1.2421875, 1.0048828125, -0.47265625, 0.5205078125, 0.88232421875, 0.2734375, -0.71533203125, -0.317138671875, -0.693359375, 1.091796875, 1.3388671875, -2.1171875, -1.205078125, 2.798828125, 0.1497802734375, 0.65771484375, -0.2183837890625, -1.26953125, 0.1865234375, 0.9892578125, -1.8564453125, -0.28466796875, 0.2293701171875, -0.158203125, -0.47119140625, -2.0078125, -0.277099609375, -0.1446533203125, -0.818359375, 0.2724609375, -0.2822265625, -1.2587890625, 0.29296875, 0.255859375, -1.619140625, 0.50390625, 2.720703125, -1.08984375, 0.8037109375, 0.6630859375, 1.318359375, -0.355224609375, 0.45458984375, -1.439453125, -1.5166015625, 1.46484375, -0.362060546875, -0.673828125, -1.0693359375, -0.09375, 1.3427734375, -1.1767578125, 0.8017578125, 0.489501953125, 1.21875, 0.2244873046875, -1.041015625, -1.6025390625, 0.724609375, 0.8671875, -0.54150390625, -1.689453125, 0.8271484375, -0.52685546875, 0.28662109375, -1.5849609375, -1.1689453125, -0.55517578125, 0.64453125, -0.302734375, -1.05078125, -0.28857421875, -0.05853271484375, 1.265625, -0.78955078125, -0.1307373046875, -0.386962890625, 1.7109375, 2.427734375, 1.765625, -0.165771484375, -0.74951171875, -1.302734375, 0.3486328125, 0.63134765625, 1.1171875, 0.03204345703125, -0.83544921875, -0.4130859375, 0.76123046875, 0.27099609375, -0.68408203125, 0.4677734375, -0.251953125, 0.7041015625, -0.11346435546875, -0.387939453125, 1.9208984375, -0.7216796875, 1.6640625, 0.63525390625, 1.6845703125, 1.36328125, -0.11163330078125, -0.84716796875, -0.76953125, 0.6484375, 0.403076171875, 0.270751953125, 0.97021484375, 0.517578125, -0.93212890625, -0.1217041015625, 0.890625, 0.29638671875, -0.027984619140625, -0.51806640625, 0.9833984375, -2.509765625, -1.5009765625, 1.2861328125, -0.126708984375, -0.68115234375, -0.2225341796875, -0.75146484375, 0.3046875, -0.1756591796875, 0.8935546875, 1.197265625, -2.521484375, -2.66015625, -0.47705078125, 0.40283203125, -0.396728515625, 0.458251953125, -1.13671875, -0.35595703125, 0.263671875, 1.08984375, -2.623046875, -0.533203125, 0.69287109375, -2.173828125, -1.1025390625, -3.34765625, -0.2197265625, 0.5888671875, 0.376953125, 0.6025390625, -0.2423095703125, 0.31298828125, 0.32421875, -1.27734375, -0.0958251953125, -0.953125, 0.603515625, 0.06829833984375, -1.083984375, -0.2442626953125, 0.96240234375, -0.427490234375, 0.037506103515625, 0.56787109375, 2.2421875, 1.7119140625, 1.029296875, 0.394775390625, 0.385986328125, -0.432861328125, 0.1329345703125, -1.1123046875, 1.51171875, 0.9619140625, -2.611328125, 0.087158203125, -0.490234375, 1.15234375, -1.3359375, -0.6630859375, 0.63330078125, 0.10791015625, -2.4765625, -0.319091796875, -0.2303466796875, 0.85302734375, 1.732421875, 0.541015625, -0.3134765625, 0.113037109375, -1.6298828125, -0.2900390625, -0.269287109375, 1.208984375, 1.16796875, -0.324951171875, 0.8505859375, 0.06866455078125, 0.010955810546875, 0.004451751708984375, 0.572265625, 0.86572265625, 3.14453125, 0.229736328125, 0.99267578125, 0.62109375, -0.57666015625, 0.73876953125, -0.044158935546875, -0.061126708984375, -0.14453125, -0.4609375, 0.1175537109375, -1.4453125, -1.037109375, 0.98193359375, -0.23876953125, 0.78125, -2.123046875, 1.255859375, -0.378173828125, -0.0223388671875, -0.6474609375, 0.033447265625, -0.7607421875, -0.302734375, 1.4306640625, -1.2578125, -0.395751953125, -1.7744140625, -1.1298828125, 0.7744140625, -0.292236328125, 0.414794921875, 1.7216796875, -0.438232421875, -1.9755859375, 1.5419921875, -0.60693359375, 0.010345458984375, 0.3447265625, -0.392822265625, 1.978515625, 0.302490234375, 0.6533203125, -0.6728515625, 0.33544921875, -0.2275390625, 0.499267578125, 0.0704345703125, 0.052001953125, -1.62109375, -2.40234375, 0.76171875, -0.3115234375, 0.274169921875, -2.1796875, -1.0439453125, -0.2435302734375, -0.276123046875, 1.681640625, 0.88232421875, 0.26611328125, 0.53515625, 0.1644287109375, -0.442138671875, 0.7001953125, 0.43603515625, 2.50390625, -0.346923828125, -0.68359375, -1.0087890625, -0.34130859375, -0.0228424072265625, 0.2078857421875, 0.408935546875, -0.7373046875, -0.338623046875, -1.3232421875, 1.0888671875, 0.378662109375, -0.0885009765625, 0.2281494140625, -0.01168060302734375, -1.7861328125, 0.271728515625, -0.3203125, 0.299072265625, 1.078125, 0.8076171875, 0.070068359375, 0.65185546875, -0.368408203125, -0.4794921875, -0.5263671875, -0.1654052734375, -0.5244140625, 0.52392578125, 0.452392578125, -0.208740234375, -0.073974609375, -0.48388671875, -0.368408203125, 0.388671875, -1.150390625, -0.5595703125, 0.91259765625, -0.081298828125, -1.7294921875, -0.275390625, 0.708984375, -0.74169921875, 0.291748046875, -0.7880859375, 0.1802978515625, 0.48193359375, 0.169677734375, 0.1641845703125, -1.462890625, -0.2462158203125, 0.0577392578125, -0.459228515625, -0.07843017578125, 0.873046875, 0.41748046875, 1.548828125, 0.56787109375, 1.8125, -1.0458984375, 0.8505859375, -0.017547607421875, -0.251220703125, 0.003101348876953125, 1.673828125, 0.6806640625, -0.54150390625, 0.9287109375, 1.4150390625, -1.779296875, -0.464599609375, -2.05078125, 0.1519775390625, -2.04296875, 0.07342529296875, 0.405029296875, -0.54443359375, 1.15625, 0.873046875, 0.9130859375, 0.5888671875, 0.71875, -0.83447265625, -0.253173828125, 0.243896484375, 2.201171875, 0.2120361328125, 0.186767578125, -1.3359375, -0.5732421875, -1.0126953125, 1.2744140625, 1.2900390625, 0.876953125, 2.169921875, 0.71240234375, 0.54833984375, -1.244140625, -0.75244140625, 0.6953125, 0.05145263671875, 0.88330078125, -1.587890625, 0.4267578125, -0.642578125, 0.82080078125, 0.1683349609375, -0.424560546875, 0.2191162109375, 1.677734375, 1.552734375, 0.5947265625, -0.5078125, 0.7626953125, 0.2666015625, -1.3125, -0.396728515625, 0.8525390625, -0.89990234375, -0.145263671875, -0.20654296875, 0.1412353515625, 0.464111328125, 0.0269927978515625, -1.5947265625, 0.1929931640625, 0.1744384765625, -1.59375, -0.298583984375, 0.035491943359375, 2.17578125, 0.54833984375, 1.5400390625, 0.045654296875, 0.098388671875, -0.39111328125, -0.85498046875, 0.09197998046875, -0.352783203125, -0.2457275390625, -1.5478515625, 0.4619140625, -0.52294921875, -1.1591796875, 0.685546875, 0.256103515625, -0.0171051025390625, 0.313232421875, 0.425537109375, -1.6591796875, 0.36572265625, -0.5771484375, 0.6328125, 0.875, 1.40625, 0.703125, 1.4541015625, -0.6923828125, -0.125244140625, -1.1767578125, 1.2294921875, 1.259765625, 0.21923828125, -0.97509765625, -0.334228515625, 0.505859375, -1.3330078125, -1.015625, 0.047576904296875, 0.5126953125, 1.126953125, 0.1934814453125, -0.3681640625, -0.2489013671875, -1.5361328125, 0.239990234375, -0.2215576171875, 0.0888671875, -0.56396484375, 0.15380859375, 1.4716796875, -2.052734375, 0.7861328125, -1.7001953125, -1.0791015625, -1.1630859375, 0.50439453125, 0.0916748046875, 1.2919921875, 1.302734375, 0.349365234375, 0.08001708984375, 0.95166015625, 1.13671875, -1.7197265625, -0.8212890625, 1.4501953125, 0.3662109375, -0.7236328125, -0.06085205078125, 0.94921875, -1.658203125, -0.68994140625, -0.344482421875, -0.6943359375, 0.62060546875, 0.97705078125, 0.1055908203125, -1.0986328125, -0.5625, -1.447265625, 0.60595703125, -0.1412353515625, -0.1331787109375, -0.1317138671875, -0.51611328125, 0.8828125, -0.0311737060546875, -1.44921875, 0.2244873046875, 0.291015625, -0.587890625, 1.8642578125, 0.479248046875, -0.2587890625, -0.293212890625, -1.4775390625, 0.72509765625, 1.296875, 0.7060546875, 0.63671875, -0.38818359375, 0.399169921875, -0.406494140625, -0.0418701171875, 1.525390625, 1.9599609375, -0.397705078125, -1.3916015625, 0.615234375, -1.595703125, -0.89697265625, -1.193359375, -0.6240234375, -1.259765625, -1.4404296875, -0.6591796875, 0.062286376953125, -0.2337646484375, 1.6611328125, -0.78955078125, 0.02850341796875, 0.325439453125, 1.2841796875, 0.2432861328125, -0.1829833984375, -1.314453125, 2.572265625, 1.478515625, -2.302734375, -0.366455078125, -0.19677734375, -0.43408203125, -0.275634765625, 1.703125, 0.0150299072265625, 0.0533447265625, 0.274169921875, 0.93310546875, -1.376953125, 0.1978759765625, 0.54248046875, -0.07171630859375, -0.2427978515625, 0.1243896484375, 0.072265625, -0.406494140625, -0.63623046875, 2.291015625, -1.046875, -1.6240234375, -1.8466796875, -1.470703125, -0.958984375, 0.08514404296875, -0.78759765625, 1.2646484375, -0.9501953125, 0.47705078125, -1.0498046875, 0.310546875, -1.1259765625, 0.83935546875, -0.1510009765625, -0.6865234375, -1.0849609375, 1.111328125, -0.78173828125, -1.568359375, 0.11004638671875, -0.0244598388671875, 0.0172119140625, -0.71240234375, -0.2161865234375, 0.312255859375, 1.578125, 1.6640625, 1.9580078125, 0.0033206939697265625, -0.1400146484375, 0.35986328125, -0.0394287109375, -0.853515625, -0.052154541015625, -1.4921875, 0.0143890380859375, -1.38671875, -0.497314453125, 0.50634765625, -0.443603515625, -1.70703125, 1.09375, -0.62158203125, 0.191650390625, -0.732421875, 0.8203125, -0.489990234375, -1.1201171875, 0.348388671875, 0.7109375, 0.00412750244140625, -0.859375, -0.54541015625, -0.139404296875, -0.4501953125, -0.71435546875, -0.50537109375, -1.4541015625, -0.0760498046875, -1.1572265625, 0.77685546875, 0.390625, -0.76123046875, 0.1888427734375, -1.4052734375, 0.74072265625, -1.1083984375, 2.05078125, -2.15625, -0.74267578125, -1.0087890625, 0.98828125, -0.0181121826171875, 0.07177734375, 0.419921875, 0.88134765625, -0.552734375, -0.426513671875, 0.0131378173828125, 0.01800537109375, -0.222900390625, -1.21484375, -0.3681640625, 0.22412109375, 0.54736328125, 0.351318359375, 1.072265625, 0.0010128021240234375, 0.51904296875, 0.281494140625, -0.43359375, -0.2802734375, -1.6220703125, 1.2138671875, 0.269775390625, -0.6572265625, -0.2626953125, -0.833984375, 1.1171875, -0.59326171875, 1.916015625, 1.427734375, 0.81689453125, 1.5341796875, -0.001956939697265625, -0.47607421875, -0.76171875, 1.216796875, 0.386474609375, -0.890625, -1.390625, -0.888671875, 0.88916015625, -2.365234375, 0.15283203125, -0.0221405029296875, -0.490966796875, -0.5576171875, 0.90966796875, -0.416748046875, 0.67431640625, -0.62890625, -0.54345703125, 0.2423095703125, -1.21484375, -0.254638671875, -1.19921875, 0.50732421875, -0.64892578125, -1.2470703125, 0.018463134765625, 0.59423828125, -1.7392578125, -1.7236328125, -1.580078125, 0.39453125, 1.4130859375, -0.43701171875, 0.71337890625, -0.150634765625, -0.106689453125, -1.6650390625, -1.5478515625, 0.912109375, 0.56787109375, 0.462890625, 1.4013671875, 1.09375, -0.1424560546875, 1.7421875, -1.05859375, 0.5634765625, -0.84423828125, 0.82568359375, 1.451171875, -1.3427734375, -0.2186279296875, -0.1900634765625, 1.4501953125, -0.7548828125, 1.28515625, 0.07684326171875, 0.1400146484375, 1.953125, -1.0244140625, -0.177734375, -0.9951171875, -0.356201171875, 1.984375, -1.1513671875, 0.42333984375, -0.87451171875, -0.1905517578125, -0.88720703125, -1.4931640625, -0.84326171875, -0.8095703125, -0.351318359375, -1.1650390625, 0.367919921875, -0.58154296875, -0.355224609375, 0.9296875, 1.24609375, 0.08056640625, 0.455078125, 1.955078125, 1.3369140625, -0.89794921875, -0.93505859375, -0.53662109375, -0.63623046875, -0.042755126953125, -0.138427734375, 0.61572265625, -0.0926513671875, 0.50439453125, -2.265625, 1.4619140625, -0.0838623046875, -0.08868408203125, -0.337646484375, 0.421142578125, -0.284423828125, 0.206298828125, -0.08807373046875, -1.951171875, -0.345703125, 0.5361328125, -0.18310546875, 0.7646484375, 1.068359375, 0.7255859375, -0.283203125, -1.3427734375, -1.2578125, 1.2529296875, 0.8603515625, -0.168212890625, 0.00547027587890625, -2.591796875, -1.4453125, -0.54248046875, 0.16015625, -0.35400390625, 0.90771484375, -0.49951171875, 0.1134033203125, -0.1365966796875, -0.408447265625, -2.283203125, 1.197265625, 1.212890625, -1.0712890625, 0.5849609375, -0.405029296875, -1.42578125, 1.619140625, 0.09063720703125, -0.213134765625, -2.076171875, -0.7626953125, 0.40380859375, 1.4716796875, 0.5048828125, 0.115966796875, 0.5810546875, 0.337646484375, 1.6796875, -1.6875, 0.337890625, -0.5400390625, 0.49658203125, -0.5009765625, -0.05731201171875, 0.56689453125, 1.36328125, -0.287841796875, 0.77001953125, -1.248046875, -0.44677734375, 0.93701171875, -0.83740234375, -0.025146484375, -1.1142578125, 0.315185546875, 1.0283203125, 0.264404296875, 0.047760009765625, -0.58837890625, 0.7470703125, 0.030487060546875, 1.431640625, 1.244140625, -0.67431640625, -1.85546875, -0.060211181640625, -1.2900390625, -0.2315673828125, -0.06427001953125, 0.77978515625, 0.767578125, -0.8203125, -0.54931640625, -1.146484375, 0.40673828125, -0.025238037109375, -0.286865234375, -1.4150390625, -1.6328125, -0.2125244140625, 1.3232421875, 1.583984375, 0.1376953125, -0.059539794921875, -0.354736328125, -0.67431640625, 0.1422119140625, 0.325927734375, 0.279541015625, -0.59423828125, -0.70361328125, -0.72900390625, 0.306640625, -1.6904296875, 0.32373046875, 1.1240234375, -0.37451171875, -0.11273193359375, 1.072265625, -0.80810546875, -0.5419921875, -0.13232421875, 0.4951171875, -0.2161865234375, -0.99267578125, -0.4296875, -0.74951171875, -1.193359375, -0.93896484375, -0.5595703125, 1.4755859375, -0.58203125, 2.1796875, 2.111328125, 1.875, 0.20458984375, -0.61669921875, 0.210693359375, -1.0693359375, 1.908203125, 0.63818359375, -0.73974609375, 0.0293426513671875, 1.603515625, -1.0859375, 0.01415252685546875, 0.224609375, 0.1591796875, -0.11016845703125, 0.1343994140625, -0.60888671875, -1.822265625, -1.01953125, 0.17529296875, -0.95654296875, 1.8076171875, -0.490966796875, -0.75048828125, 1.0810546875, -0.296142578125, -0.474609375, -1.4462890625, 0.304931640625, -0.2313232421875, -1.1142578125, -1.5703125, -0.958984375, -0.226318359375, 1.2685546875, -0.5302734375, 1.0380859375, -0.099609375, -1.1474609375, 0.201171875, 1.7490234375, 1.896484375, 0.7041015625, -0.62353515625, -0.9912109375, 1.4814453125, -1.361328125, 1.9052734375, -0.30224609375, 0.301513671875, 1.3251953125, -0.435546875, -0.3720703125, 1.3642578125, 2.662109375, 1.5830078125, -0.296142578125, 0.74951171875, -0.173095703125, 0.4658203125, -2.263671875, -0.07257080078125, 1.1162109375, 1.8515625, -2.3203125, 0.1622314453125, 0.191650390625, 0.267822265625, -0.50439453125, -1.0771484375, 1.0771484375, 0.328857421875, 1.08203125, 1.0380859375, -0.3876953125, 0.52880859375, 1.0146484375, 0.30908203125, -0.07281494140625, -1.2744140625, 0.51806640625, -1.41796875, 0.52783203125, -0.35986328125, 0.051788330078125, 0.720703125, -1.0, 0.046234130859375, 0.11114501953125, 0.80810546875, -0.10125732421875, -0.252685546875, 0.392333984375, -0.1971435546875, -0.07470703125, -1.09375, 0.1080322265625, 1.2255859375, 0.63720703125, -0.94384765625, -1.578125, -2.630859375, 2.8359375, -0.52880859375, -0.501953125, 0.4814453125, -0.011810302734375, -0.2034912109375, 0.59326171875, 2.23046875, -0.391357421875, -0.17431640625, -0.25390625, -0.8232421875, 0.215087890625, 0.55615234375, -0.87890625, 0.323486328125, 2.76171875, 0.218994140625, 0.55078125, 0.77490234375, -0.328125, 0.204833984375, -0.1195068359375, 0.8154296875, 1.5146484375, 0.256591796875, 0.96044921875, -1.8564453125, 0.845703125, 0.974609375, -0.42822265625, -0.368408203125, 0.7041015625, -0.6513671875, -1.265625, -0.1773681640625, -0.1187744140625, -0.99755859375, 0.5576171875, 1.462890625, 0.69287109375, 0.0379638671875, -2.234375, -1.4052734375, -0.89599609375, 0.5966796875, 1.341796875, -1.6484375, 1.9599609375, -0.8203125, 0.7255859375, 2.068359375, 2.201171875, 0.09637451171875, 0.35595703125, 0.442626953125, -1.1787109375, -0.09429931640625, 0.0694580078125, -0.0200347900390625, -0.53125, -0.58984375, 0.30517578125, 1.1259765625, 0.7607421875, 0.63037109375, -0.1092529296875, -1.865234375, -0.49658203125, -0.292236328125, 1.2041015625, 0.09002685546875, -0.442626953125, 1.5283203125, -0.31201171875, -0.01983642578125, 0.85791015625, 0.33740234375, -0.48388671875, 1.216796875, 1.5654296875, -0.24169921875, 0.5634765625, -0.984375, 1.4541015625, -0.962890625, 0.30712890625, -0.73291015625, 0.002384185791015625, -0.4345703125, -1.7119140625, -1.6416015625, 0.88720703125, -0.323974609375, -0.82958984375, -0.256103515625, 0.5146484375, -0.060638427734375, 0.201416015625, -0.6376953125, 0.69091796875, 0.83837890625, 0.347412109375, -0.58984375, -0.8828125, -0.414306640625, 1.03125, -0.603515625, -1.06640625, -1.3359375, 0.1385498046875, -0.79052734375, -0.83349609375, -1.435546875, 0.79541015625, 0.144287109375, 0.86279296875, -0.4130859375, -0.76611328125, 1.8515625, -0.1767578125, 0.56005859375, 0.6318359375, 0.2244873046875, 1.2109375, -0.55859375, -0.751953125, -0.7998046875, -1.0771484375, 0.2744140625, 1.7744140625, -0.9814453125, -1.6171875, 0.802734375, 0.420166015625, 2.37109375, 0.86279296875, -0.517578125, 2.22265625, -0.13427734375, 0.6611328125, -0.380859375, 0.51513671875, -2.0703125, -1.529296875, -0.426025390625, 0.78955078125, 0.98681640625, 1.57421875, 0.306396484375, 0.59033203125, 1.7197265625, 0.56005859375, 1.2373046875, 0.449462890625, 1.19921875, 1.2841796875, 1.5458984375, -0.04144287109375, -0.05194091796875, -0.11944580078125, 0.5732421875, 1.298828125, -0.0699462890625, -0.000995635986328125, 0.10491943359375, 1.060546875, 0.251953125, -0.63427734375, -1.2763671875, -0.1824951171875, 1.48828125, -0.2025146484375, 0.07562255859375, 1.1533203125, 1.51171875, -0.0294036865234375, 0.0869140625, -1.2783203125, -0.85302734375, -0.22900390625, -0.0117340087890625, 0.75732421875, -1.380859375, -1.828125, -0.8125, 0.43505859375, 0.354736328125, 0.138916015625, -0.41748046875, -1.6572265625, -1.2919921875, 0.46630859375, 0.1956787109375, -0.8974609375, -1.0712890625, -1.2265625, 0.794921875, -1.1484375, -0.95361328125, -1.3798828125, -1.0146484375, 1.3193359375, -0.2230224609375, -0.79248046875, 0.1822509765625, -1.0888671875, 0.9521484375, -1.51953125, -1.18359375, -0.80029296875, -1.029296875, 0.71044921875, -0.58544921875, 0.022705078125, 2.06640625, 2.171875, 0.9169921875, -0.98681640625, 0.455810546875, -0.505859375, -1.0634765625, 0.7666015625, 0.0748291015625, 1.2392578125, 0.6083984375, -1.8271484375, 1.501953125, 0.418701171875, -0.5234375, 2.044921875, -0.595703125, -0.2454833984375, -0.7802734375, -0.166015625, 0.427734375, 0.75927734375, 0.1717529296875, -1.099609375, 2.310546875, -0.462158203125, -1.7626953125, -0.96240234375, 1.337890625, 0.57470703125, 2.63671875, -2.197265625, -0.7373046875, 0.094482421875, 0.8876953125, 0.66748046875, -0.8662109375, -0.239501953125, -0.6416015625, 0.65869140625, -0.1129150390625, 1.693359375, 1.1552734375, 1.91796875, 0.845703125, -0.4951171875, 1.150390625, -0.755859375, -1.04296875, 0.461181640625, -0.66455078125, 1.0439453125, -0.211669921875, 2.22265625, -0.70654296875, 0.54248046875, -0.09539794921875, 1.189453125, 0.8271484375, -1.3740234375, -0.0458984375, 1.2470703125, 0.78857421875, -1.373046875, 0.489013671875, 0.11444091796875, 1.6357421875, 1.7158203125, 1.04296875, -0.4287109375, 0.387939453125, -0.83837890625, 0.2156982421875, 0.384521484375, -0.030364990234375, -0.2408447265625, -0.70166015625, -0.363525390625, -1.7421875, 1.0888671875, -0.69775390625, -1.9287109375, 1.4716796875, 0.67041015625, -0.8740234375, 0.705078125, -0.09527587890625, -0.417724609375, 0.2822265625, -0.181884765625, -0.9150390625, 0.07171630859375, 0.63134765625, -2.35546875, 0.62255859375, 0.466064453125, -0.438232421875, 0.0528564453125, -0.5595703125, -1.4248046875, 1.7900390625, -1.5048828125, 0.1912841796875, 2.712890625, 0.164794921875, -0.75927734375, -1.673828125, -0.63037109375, -0.517578125, -0.0028476715087890625, -1.7734375, -0.54931640625, 0.22607421875, 1.0458984375, 1.53515625, -1.181640625, -0.291259765625, 1.390625, -0.88134765625, -1.044921875, -1.548828125, -1.7255859375, -0.11529541015625, -0.06060791015625, -0.1502685546875, 1.6240234375, 0.09332275390625, -1.12890625, 0.552734375, 0.927734375, -0.28369140625, 1.3544921875, -0.58642578125, 0.33447265625, 0.1971435546875, 0.4521484375, -0.422607421875, -0.423095703125, -0.238037109375, 0.28662109375, 1.2138671875, -0.16845703125, -0.50439453125, 2.439453125, 0.6943359375, -0.041290283203125, 0.295166015625, 1.05859375, -0.2313232421875, -0.483642578125, 0.01104736328125, -0.88623046875, -1.3330078125, 0.81396484375, 0.27490234375, -0.29150390625, -2.767578125, 1.2138671875, 0.53271484375, 0.76025390625, -0.78466796875, 0.0826416015625, 1.462890625, 0.171142578125, 0.23974609375, 1.1005859375, 0.3671875, 0.2685546875, 0.273193359375, -1.20703125, 2.177734375, -0.923828125, 0.537109375, 1.9775390625, -0.220458984375, -0.12646484375, -0.45263671875, -0.498779296875, 1.732421875, -0.51953125, 0.44287109375, -1.1201171875, -1.380859375, 0.93896484375, 0.220458984375, 1.5634765625, -0.65380859375, -0.4619140625, 0.67919921875, -1.435546875, 1.111328125, 1.279296875, -0.11224365234375, 1.294921875, 0.1953125, -2.189453125, 2.787109375, 0.005413055419921875, 0.415771484375, 0.37646484375, -0.5546875, 0.7998046875, 1.2099609375, 0.85498046875, 0.1036376953125, -0.314208984375, 0.72412109375, 0.80859375, 0.8876953125, -0.173828125, -1.6669921875, 0.8154296875, 0.6083984375, -0.748046875, -0.1280517578125, 0.6162109375, 0.0129241943359375, -0.036468505859375, -1.4365234375, 0.029815673828125, -0.84912109375, 0.6142578125, -0.447021484375, 1.009765625, 0.755859375, -1.3330078125, 0.6025390625, -1.0419921875, -0.16650390625, 0.9423828125, 0.3427734375, 2.86328125, 1.3173828125, -0.274169921875, -0.325439453125, 0.84765625, 1.310546875, 0.7685546875, 1.98046875, 0.349365234375, -1.2314453125, -1.3330078125, -1.4912109375, 0.255126953125, -0.748046875, 1.9130859375, 0.300537109375, 1.31640625, 2.087890625, -0.28857421875, 0.76123046875, -0.49072265625, -0.62109375, -2.6484375, 0.5244140625, 1.732421875, -0.71875, -0.76806640625, 0.64990234375, -0.76904296875, 0.24267578125, 1.34765625, 0.154296875, -0.7529296875, -0.497314453125, -0.432861328125, -0.7705078125, -1.3203125, -0.60400390625, 0.403076171875, 0.455078125, 0.62939453125, -1.0283203125, -1.5751953125, -0.0308990478515625, -0.350830078125, 1.3203125, 1.578125, 0.477783203125, 0.0938720703125, 0.04241943359375, -0.5439453125, -0.74658203125, -0.876953125, 1.005859375, 1.314453125, 0.794921875, 1.2890625, 0.177001953125, -1.2529296875, 0.3046875, 0.2548828125, -1.05078125, 0.039093017578125, -0.0217437744140625, 0.7216796875, 2.14453125, 1.390625, 0.31884765625, -1.07421875, -1.0712890625, 0.064453125, -0.09954833984375, 0.95556640625, -0.491455078125, -1.28515625, -0.9951171875, -2.083984375, 0.8349609375, -1.0712890625, -0.005950927734375, -1.3330078125, -0.2037353515625, 0.368896484375, 0.0411376953125, -1.8173828125, 0.357666015625, -0.86572265625, -1.3994140625, -1.3955078125, -0.263671875, -1.23828125, 0.89453125, 1.0302734375, 0.96923828125, 1.8056640625, 1.0947265625, 0.439697265625, -0.82177734375, 1.8740234375, 0.299072265625, 1.3427734375, 1.119140625, 0.64453125, -1.087890625, -0.7744140625, -0.259033203125, -1.0673828125, 1.318359375, -0.55810546875, -0.4033203125, 0.771484375, 1.560546875, -0.66455078125, -0.57421875, -0.759765625, 1.2275390625, -0.201416015625, 0.7255859375, -0.4296875, 0.300537109375, 0.0445556640625, 0.81640625, 0.494140625, -0.0300140380859375, 0.2427978515625, -0.08050537109375, 2.904296875, 0.380859375, -1.65234375, 0.267333984375, 1.05078125, -2.541015625, -1.0615234375, -0.8095703125, 0.93310546875, 0.248779296875, 0.5517578125, -0.213134765625, 1.056640625, -0.43115234375, -0.10540771484375, -0.205322265625, 0.402099609375, 0.480224609375, -0.57568359375, 0.13134765625, -0.5771484375, -0.45458984375, 0.29736328125, -1.517578125, -0.461181640625, 1.4580078125, -0.35888671875, 0.4677734375, 0.78955078125, 2.556640625, -2.357421875, 0.05926513671875, 0.57763671875, -0.40478515625, 0.318603515625, -1.9013671875, -1.51171875, -0.82763671875, -0.410400390625, 0.261474609375, -0.335205078125, 0.15185546875, -0.00897216796875, 0.1295166015625, 0.06610107421875, 2.072265625, 0.48583984375, 0.31591796875, 0.37109375, 0.39404296875, -0.439453125, -0.36767578125, -0.132080078125, -0.387451171875, -0.409423828125, -2.09765625, -1.8974609375, 0.6064453125, -1.5947265625, 1.2958984375, 0.71142578125, -1.150390625, 1.71484375, 0.89794921875, -1.599609375, 1.06640625, 0.33251953125, 1.85546875, 1.126953125, -0.7607421875, 0.74658203125, -1.544921875, -0.218017578125, 0.65283203125, 1.1748046875, 1.5634765625, 0.057220458984375, -2.1953125, 0.372314453125, 1.3017578125, -0.1859130859375, -1.7216796875, -0.01372528076171875, -0.83056640625, -0.11065673828125, 0.425537109375, 1.5126953125, -0.2281494140625, -0.939453125, 0.443359375, -1.8203125, 0.32373046875, -1.080078125, -0.391357421875, -0.65380859375, -1.2470703125, 0.494140625, -1.0556640625, 0.17724609375, 0.388427734375, 1.1376953125, 0.1934814453125, -0.1529541015625, 1.3173828125, -0.89306640625, -0.43603515625, 0.59033203125, -2.05859375, 0.70166015625, 1.4287109375, 0.2958984375, -1.419921875, -0.041839599609375, -0.191650390625, 0.061737060546875, -0.315673828125, -1.2021484375, 0.07586669921875, 1.3046875, -0.98876953125, -1.8193359375, 0.61474609375, -0.51025390625, -0.2822265625, 0.7763671875, 0.95458984375, 0.2890625, 0.958984375, -0.288818359375, -0.09478759765625, 0.50341796875, 0.36962890625, -0.12188720703125, 1.291015625, -1.5078125, 0.50146484375, 0.78564453125, -0.013427734375, -0.76025390625, -0.74951171875, -0.5771484375, 0.460693359375, -0.6171875, 0.5673828125, 0.056427001953125, -0.418212890625, -1.337890625, -0.6630859375, -1.42578125, -0.304931640625, 1.8037109375, -1.4189453125, 1.8916015625, -0.454833984375, -0.456298828125, 0.7802734375, 0.9404296875, 0.99609375, 1.4619140625, -0.55908203125, 0.1092529296875, 1.1328125, -0.2607421875, 0.69970703125, 0.02099609375, 0.354736328125, -1.5380859375, 0.751953125, 0.54833984375, -1.0576171875, -1.2021484375, 0.257080078125, -0.88720703125, 1.1796875, -1.7841796875, -0.81298828125, 0.1915283203125, -1.0888671875, 0.6083984375, -0.458740234375, 1.568359375, -1.0556640625, 1.6923828125, 0.966796875, 0.73583984375, -0.09149169921875, -0.5693359375, 0.18310546875, 0.92236328125, 1.130859375, 0.8125, 1.49609375, 0.99365234375, 1.7880859375, -0.96630859375, 0.7109375, -1.8935546875, -0.55078125, 1.26953125, -1.212890625, 1.2587890625, 2.64453125, -1.97265625, -0.59619140625, -0.45458984375, 0.2418212890625, -1.263671875, 0.347412109375, 0.697265625, -0.63818359375, 0.70556640625, -0.35791015625, 1.0078125, 0.1646728515625, 0.96142578125, -0.72705078125, 0.6435546875, -0.71337890625, -1.23046875, -0.86328125, 2.07421875, 0.410888671875, -0.56982421875, 0.486083984375, -0.281982421875, 1.134765625, 2.125, -0.01568603515625, 0.8564453125, 1.923828125, 0.232177734375, -0.20166015625, -0.2763671875, 0.6552734375, -1.482421875, -0.58154296875, -0.91552734375, 1.5380859375, 0.81201171875, 0.450927734375, -1.0927734375, 1.044921875, 0.65966796875, -0.63134765625, 1.103515625, -1.8115234375, -0.43603515625, 1.5400390625, -1.0107421875, -0.58984375, -0.14306640625, -1.4541015625, 1.3154296875, 0.113525390625, 0.004669189453125, -0.92822265625, 0.95654296875, -1.6357421875, -1.4697265625, -0.376953125, 0.284912109375, 1.2529296875, 1.1982421875, -0.165283203125, 0.4970703125, -0.64501953125, 1.3134765625, -1.3515625, 0.270263671875, 0.294677734375, 0.1678466796875, 2.818359375, -0.01198577880859375, -0.650390625, -0.677734375, 0.04217529296875, -0.5029296875, -0.5234375, -0.65087890625, 0.93408203125, -1.572265625, -0.39306640625, 1.095703125, -0.203369140625, -0.5087890625, -0.5205078125, -0.68994140625, -0.462158203125, 2.08984375, -0.330810546875, -2.29296875, 0.07806396484375, -1.4658203125, -0.10919189453125, 0.08038330078125, -1.2744140625, 0.93408203125, -0.25537109375, 1.0185546875, -0.1478271484375, -0.1951904296875, 0.379150390625, -0.697265625, 0.7177734375, -2.033203125, -0.77978515625, 2.1484375, -0.73876953125, -0.17529296875, -0.55029296875, 0.2646484375, -1.2509765625, 0.0225982666015625, 1.0986328125, -0.357421875, 0.080810546875, 1.5439453125, 0.67578125, -0.02471923828125, 0.6201171875, -0.79345703125, -0.169677734375, -0.71484375, 0.182373046875, 0.1358642578125, 0.55517578125, -1.244140625, 0.07769775390625, -1.4951171875, -0.84423828125, 1.0302734375, -0.52294921875, 1.3232421875, -0.67041015625, -1.251953125, 0.01132965087890625, 0.17333984375, 0.68115234375, -1.2880859375, 0.36279296875, 1.587890625, -2.2265625, -0.28466796875, -0.5927734375, -0.51025390625, 0.673828125, -0.18603515625, 1.4296875, 0.52734375, -1.3076171875, 1.05859375, -0.779296875, 1.1455078125, 1.029296875, -0.09759521484375, -1.2734375, 0.1533203125, -1.1904296875, 1.1826171875, 0.794921875, 0.83740234375, 1.6494140625, 0.267578125, 0.94384765625, -0.452392578125, -0.62109375, 1.119140625, 0.638671875, 1.1904296875, -0.281982421875, -0.5048828125, -0.90869140625, -0.94189453125, -0.4658203125, -0.319091796875, -0.6103515625, -1.9033203125, 0.31640625, 0.80419921875, 2.294921875, -1.7021484375, -0.5126953125, 1.677734375, 1.5595703125, 1.23828125, 0.04119873046875, 0.50634765625, -0.07818603515625, 0.2158203125, -1.5107421875, -0.8857421875, 1.125, -1.0888671875, -0.4814453125, 0.436767578125, 0.671875, 0.5927734375, 0.97265625, 0.1846923828125, 0.359375, 0.52587890625, -1.38671875, 1.12890625, 0.576171875, -1.849609375, -0.135986328125, 0.55712890625, 0.163818359375, -1.0302734375, -0.75732421875, 0.25390625, -0.97802734375, 0.68408203125, 1.177734375, -0.385498046875, 2.58984375, 0.8115234375, 0.53759765625, -1.341796875, -0.095703125, -0.2159423828125, 0.15576171875, -2.43359375, 1.10546875, 0.26708984375, -1.81640625, 0.30810546875, 1.1201171875, 0.293212890625, 2.10546875, -0.2196044921875, 1.169921875, -0.026214599609375, 0.64404296875, -0.07647705078125, -0.0062408447265625, -1.1904296875, -1.2978515625, -0.478271484375, -3.1328125, -1.109375, -0.239013671875, 0.26318359375, -0.211181640625, 0.59423828125, 0.53955078125, -0.75927734375, 0.736328125, -0.44921875, -0.96875, 0.1278076171875, -2.5078125, -0.260498046875, 0.44970703125, 1.10546875, -0.1552734375, -1.4853515625, 1.1201171875, -0.6474609375, 0.423828125, -1.70703125, 0.556640625, -0.62109375, 1.611328125, -1.94921875, 0.529296875, 0.007427215576171875, 1.8720703125, 2.06640625, 0.108642578125, 0.3056640625, -1.197265625, -0.16455078125, 0.38232421875, -0.1484375, -0.007091522216796875, -1.95703125, 0.9755859375, 0.9755859375, -1.3427734375, 0.0958251953125, -1.619140625, -0.51904296875, -0.68408203125, -1.85546875, 0.291015625, -0.5625, -0.83642578125, -1.0478515625, -0.22509765625, 0.2161865234375, -0.200927734375, 0.3349609375, 0.25390625, 0.266357421875, 1.09765625, -0.0167236328125, 2.62109375, 1.84765625, 0.1697998046875, 0.297607421875, -0.53271484375, 0.341064453125, 1.669921875, 0.7841796875, -0.7607421875, -0.368408203125, 1.0908203125, -1.119140625, 0.296875, -0.2352294921875, 0.61474609375, -1.013671875, -0.94873046875, -1.1728515625, 1.1318359375, -0.286865234375, -0.52880859375, -0.346923828125, 0.409912109375, -0.54541015625, -0.9677734375, -0.17236328125, -1.171875, -0.97216796875, -0.19970703125, -0.8330078125, 0.95947265625, -0.24169921875, 0.9560546875, -0.466552734375, -1.419921875, -2.240234375, 0.97900390625, -0.65478515625, 0.1395263671875, 1.779296875, 0.71435546875, -2.8671875, 1.046875, -1.2626953125, -0.9365234375, -0.61572265625, 0.9970703125, -0.1356201171875, 0.339599609375, -0.89013671875, 0.83740234375, 0.34326171875, 0.08233642578125, 0.46923828125, 0.1898193359375, -0.1258544921875, 1.125, -1.0107421875, -0.360595703125, 1.4013671875, 0.58349609375, 1.8916015625, -1.4736328125, -1.388671875, 1.380859375, -0.466552734375, 0.301513671875, -1.5087890625, 0.9091796875, -2.107421875, -0.4326171875, 1.4521484375, -0.8154296875, -2.251953125, 0.09893798828125, -0.27490234375, -0.74609375, -0.174072265625, -2.431640625, 0.9501953125, -0.376708984375, 0.495361328125, 0.7060546875, -0.9482421875, 1.4833984375, 0.31201171875, 0.32275390625, 1.08984375, -0.0899658203125, 0.64453125, -0.324951171875, 1.3876953125, -0.25830078125, -0.483642578125, 0.047088623046875, 0.9111328125, -0.00522613525390625, -1.2138671875, 2.107421875, 2.138671875, -0.30712890625, 1.796875, -0.14453125, 2.568359375, -0.728515625, -0.75927734375, 1.1962890625, -0.1700439453125, 0.9462890625, -0.50634765625, 0.1044921875, 0.9013671875, 1.7236328125, -1.9951171875, 0.53369140625, 0.91796875, -0.01335906982421875, -1.5615234375, -1.029296875, 0.9990234375, -0.6025390625, -0.68896484375, 0.016448974609375, -0.38671875, -0.6015625, -0.638671875, -1.203125, -0.59814453125, -0.60693359375, 0.1439208984375, 0.403076171875, -1.47265625, 1.20703125, -1.5625, -0.395751953125, -0.86865234375, -0.17138671875, 0.87744140625, -1.4990234375, -0.6318359375, -0.6162109375, 0.353515625, 0.30517578125, 1.5400390625, -0.0268096923828125, 0.625, 0.65966796875, -0.325927734375, 1.83984375, 0.84033203125, 1.578125, 1.119140625, 0.36376953125, -0.25341796875, -0.509765625, -0.01837158203125, 0.06695556640625, 1.671875, -1.814453125, -0.84912109375, -0.56103515625, -0.75048828125, 0.2376708984375, 0.62744140625, 0.22021484375, -1.0634765625, 0.7734375, 0.8974609375, -0.226806640625, -0.517578125, 0.97509765625, -0.403564453125, -0.828125, -0.67919921875, 1.28125, -0.08282470703125, 0.45361328125, -0.93212890625, 2.77734375, -0.58642578125, -0.69970703125, 1.3095703125, 2.078125, -0.016754150390625, 0.26611328125, -0.393310546875, -0.29638671875, -1.466796875, -0.81298828125, -0.1763916015625, -0.81591796875, 2.0859375, 1.3896484375, 0.1439208984375, -2.283203125, 0.488525390625, -0.302001953125, 0.65234375, -1.3388671875, 0.177001953125, -0.826171875, -0.51416015625, -0.1292724609375, -0.097412109375, 1.6240234375, 1.251953125, 0.44384765625, 0.30712890625, 2.4296875, 0.37353515625, -0.39404296875, -1.16015625, -0.58447265625, 0.26806640625, -1.3828125, 0.423828125, 0.61328125, 0.671875, -1.6455078125, -0.71240234375, -0.421630859375, 0.7509765625, 0.11895751953125, 0.0179443359375, 0.58154296875, -0.8779296875, -0.385009765625, -2.6640625, 1.474609375, 1.5869140625, -0.24462890625, 0.1474609375, -1.3759765625, -1.23828125, 0.7421875, 0.12298583984375, 0.55078125, 0.446533203125, -0.56201171875, 0.052154541015625, -0.496826171875, 0.82861328125, 0.51123046875, 0.51904296875, -0.1502685546875, 0.64453125, 1.92578125, 0.32861328125, -0.30859375, -0.40869140625, -0.327880859375, 2.947265625, -0.29443359375, 0.56689453125, -0.414794921875, 1.78515625, -0.379150390625, 0.279541015625, 1.2255859375, 0.411865234375, 1.00390625, 0.0132904052734375, -0.59619140625, -1.3623046875, 0.400146484375, 1.109375, -1.0791015625, 0.60400390625, 1.734375, 0.2410888671875, -0.32470703125, -1.0673828125, -0.1610107421875, 0.666015625, -0.7822265625, 1.9287109375, 1.2265625, 1.6513671875, -1.2265625, 0.10791015625, -0.9599609375, -1.8251953125, -1.3701171875, -1.265625, 0.291015625, -1.197265625, 1.1396484375, -0.57080078125, -0.57421875, 0.72412109375, -0.56005859375, 0.75830078125, 1.8212890625, -0.29541015625, -0.52099609375, -1.138671875, 0.1474609375, 0.896484375, 0.6376953125, -0.4794921875, -0.943359375, 0.2216796875, 0.78857421875, -0.1654052734375, 0.083251953125, 1.4951171875, 1.9775390625, -0.054443359375, 0.40966796875, 0.00724029541015625, 1.4794921875, -0.01087188720703125, 0.09210205078125, -0.61767578125, -0.99755859375, -0.3759765625, 1.0458984375, 1.6728515625, -0.5771484375, -0.0948486328125, 0.7666015625, 1.0517578125, 0.469482421875, 1.669921875, -0.66357421875, -0.7587890625, -0.434814453125, 0.0860595703125, -0.77099609375, -0.296630859375, -0.65185546875, 0.58203125, -0.52294921875, 0.224609375, 1.65625, -0.406982421875, 0.88427734375, -0.0498046875, -0.3623046875, 0.55029296875, 0.174072265625, 0.0457763671875, -0.355224609375, -1.1015625, 0.65380859375, 1.1240234375, -1.8408203125, 1.64453125, 0.11492919921875, 0.6484375, 0.5791015625, -1.0048828125, -0.2318115234375, -0.6943359375, 0.2958984375, -1.1796875, -0.10369873046875, 0.00829315185546875, 1.4560546875, 1.41015625, -0.67236328125, -0.6474609375, 1.671875, -0.34326171875, -1.4482421875, 0.5166015625, 0.130859375, 1.1025390625, 1.28515625, 0.50927734375, -0.019744873046875, 0.447021484375, -0.174072265625, -1.3583984375, 0.1353759765625, 0.86865234375, 1.5810546875, -0.69287109375, -0.60009765625, -0.7646484375, 0.2137451171875, 1.6416015625, 0.181640625, 1.0498046875, -0.7109375, -1.2412109375, -0.779296875, -1.0, -0.066650390625, 0.039093017578125, -0.303466796875, 0.9453125, -1.212890625, 1.19140625, 0.10174560546875, -0.744140625, -2.232421875, -0.341796875, 0.48779296875, 1.95703125, -1.8583984375, 0.2415771484375, 1.537109375, 0.2081298828125, 0.56396484375, -2.474609375, 0.98388671875, 0.826171875, 0.82861328125, 0.5009765625, 0.478515625, 0.050201416015625, 0.00769805908203125, 1.455078125, -1.0771484375, 0.263427734375, 1.8466796875, -1.791015625, -0.404052734375, -1.255859375, 0.67578125, 1.166015625, 0.43212890625, -0.64404296875, 2.12109375, 0.90625, -0.12164306640625, -1.2900390625, 0.154052734375, 0.92822265625, -1.0947265625, 2.16796875, -0.1527099609375, 1.6748046875, -0.74365234375, 0.818359375, -1.0888671875, 0.24365234375, 1.0390625, -1.0048828125, 1.48046875, -0.7041015625, 0.2481689453125, 0.11712646484375, -1.216796875, 2.216796875, -0.11865234375, 0.75, 1.142578125, -0.29541015625, -0.9345703125, -0.6337890625, -0.53369140625, 0.1484375, -0.58642578125, 0.1304931640625, 1.90625, -1.0556640625, -0.266357421875, 1.17578125, -0.671875, -0.18701171875, -0.7431640625, -1.6005859375, -0.358154296875, 2.728515625, 0.61669921875, -0.26904296875, -0.99267578125, 0.149169921875, -0.70263671875, -1.080078125, 1.263671875, 0.40478515625, -0.11517333984375, 2.81640625, -0.9375, 0.94775390625, 0.473876953125, 0.9931640625, -0.3984375, 0.13232421875, 0.157470703125, -0.71630859375, -1.271484375, 0.473388671875, 0.95263671875, -0.1466064453125, 1.4384765625, 0.396484375, -0.352783203125, 0.57421875, 2.470703125, -2.244140625, 0.0262603759765625, -0.64697265625, -0.2498779296875, 1.0283203125, 0.5224609375, -0.2548828125, -0.92333984375, -1.705078125, -0.82275390625, 1.9736328125, -3.375, -0.7431640625, 0.87060546875, 2.12890625, 0.11376953125, -1.3916015625, 0.8056640625, 0.22265625, 0.2490234375, -0.16943359375, 1.3017578125, 0.90087890625, -0.402587890625, -0.1444091796875, 1.71484375, -0.96875, 1.2919921875, 0.2320556640625, -0.2061767578125, 0.93603515625, 0.328369140625, -0.7197265625, -0.0005273818969726562, 0.08074951171875, 0.529296875, 0.80322265625, -0.360107421875, 1.197265625, 0.60302734375, -0.7080078125, -0.99365234375, -0.373046875, -1.7890625, -1.353515625, 0.457763671875, -0.172119140625, -0.9306640625, 1.07421875, 0.6669921875, -0.9423828125, 0.432861328125, 1.322265625, 0.4208984375, -0.031524658203125, -0.8203125, -1.5517578125, 0.11529541015625, -1.5703125, -0.28564453125, 0.034942626953125, -0.80810546875, 0.818359375, 1.0439453125, -1.259765625, -0.564453125, 1.21484375, 1.0859375, -0.89990234375, -0.10382080078125, 0.5673828125, -0.787109375, 0.650390625, 1.6640625, 1.4736328125, -0.82421875, -0.501953125, -0.0357666015625, -0.7880859375, 0.254150390625, -0.49658203125, -1.3896484375, 2.099609375, -0.51904296875, 0.96240234375, 0.58349609375, 0.6630859375, -0.98876953125, 0.22021484375, 0.63623046875, -0.00969696044921875, -0.75341796875, 2.11328125, 0.837890625, 1.3212890625, 0.4169921875, 0.6962890625, 0.08245849609375, 1.10546875, 1.658203125, 0.5771484375, -0.97509765625, 0.216552734375, 0.6875, 0.35498046875, -0.86669921875, -2.169921875, -1.2158203125, -0.7109375, 1.021484375, -3.08203125, 1.46875, 1.5087890625, -0.2376708984375, 1.203125, -0.857421875, -0.9453125, 0.058746337890625, 0.298583984375, -0.47314453125, -0.440185546875, 0.63525390625, 2.23046875, 0.1295166015625, 0.64404296875, 0.6630859375, -0.253662109375, 0.12286376953125, 1.796875, 1.681640625, -0.1754150390625, -1.5556640625, -0.53173828125, 0.3935546875, -1.0478515625, -0.30517578125, 1.083984375, -0.70361328125, -0.366943359375, -0.9140625, 0.53955078125, -0.441162109375, -0.7626953125, -0.9404296875, 1.01953125, -0.1688232421875, -0.7158203125, 1.0458984375, -0.462890625, 0.04852294921875, -2.12109375, 0.08868408203125, -0.84814453125, 1.0869140625, -0.8076171875, -0.10821533203125, -0.54443359375, 0.81494140625, 1.63671875, 0.2484130859375, -0.49560546875, -0.60595703125, -0.865234375, -0.1927490234375, -1.0693359375, 1.423828125, 1.0556640625, -0.90087890625, 0.8037109375, 0.410400390625, -1.2998046875, -0.1658935546875, -0.564453125, 1.1083984375, -0.517578125, -0.375732421875, -0.1461181640625, 2.58203125, -1.4912109375, -0.84716796875, 0.71337890625, 0.5625, 1.3359375, 0.26123046875, 0.83984375, -0.1776123046875, 1.8330078125, 1.6953125, 0.1248779296875, -0.1026611328125, 0.6240234375, -0.56884765625, 0.5556640625, 1.021484375, 2.12109375, 1.6142578125, 1.1826171875, -1.6708984375, 0.68798828125, 1.78515625, -1.0673828125, -1.419921875, -0.1943359375, -0.263916015625, -1.357421875, -0.16015625, 0.0123138427734375, -0.3740234375, 0.397705078125, -0.88671875, -0.310302734375, -3.7578125, -0.85400390625, -0.96630859375, -0.4560546875, 2.455078125, 0.221923828125, -0.794921875, 0.1690673828125, 1.072265625, 0.85400390625, -0.03668212890625, -0.220458984375, -1.494140625, -0.70703125, -0.78173828125, 0.321044921875, 0.1175537109375, -0.189697265625, -0.01239776611328125, 1.48046875, 0.07366943359375, 0.2364501953125, -0.037017822265625, 0.1824951171875, 1.1875, 0.2802734375, 0.84912109375, -0.245361328125, 0.0019426345825195312, 1.623046875, 1.267578125, 0.591796875, 0.385986328125, -1.9677734375, 0.324462890625, 0.81884765625, -0.250244140625, 0.4482421875, 0.59326171875, -0.468994140625, -1.349609375, -1.0068359375, -0.10589599609375, -1.3212890625, -0.1968994140625, 0.576171875, 0.1212158203125, -0.402587890625, -1.0341796875, 1.34765625, 0.2386474609375, -0.3828125, 0.69482421875, 1.8876953125, 0.67138671875, -1.0947265625, 0.60791015625, -0.2337646484375, 0.83203125, -1.6787109375, 1.3642578125, -0.12322998046875, -0.36279296875, -0.026702880859375, 0.362060546875, 1.12890625, -0.70166015625, -0.09765625, 1.08203125, 1.3603515625, -0.22705078125, -0.32763671875, -1.1533203125, 0.284912109375, -1.24609375, 1.1240234375, -0.08056640625, 0.8076171875, -2.888671875, 0.33154296875, -0.305908203125, -0.95068359375, 1.208984375, 1.291015625, 0.5576171875, -0.8193359375, 0.35107421875, 0.81298828125, -0.6865234375, -0.8349609375, -1.986328125, 0.1881103515625, 0.206787109375, 0.80029296875, 0.1728515625, 0.919921875, -1.25390625, 0.80859375, -0.41748046875, -1.1044921875, -1.509765625, 0.1978759765625, 0.7392578125, 0.041259765625, -1.404296875, 0.4072265625, 0.94189453125, 0.1158447265625, -0.802734375, 1.6435546875, 0.2047119140625, -0.0019817352294921875, 0.479736328125, -0.68994140625, 0.9404296875, -1.587890625, 1.064453125, -1.48828125, 1.2333984375, 0.2222900390625, -0.005340576171875, 1.2939453125, 1.1611328125, -0.08160400390625, -1.798828125, -1.4189453125, 1.037109375, 0.68017578125, 0.93603515625, -1.279296875, 0.67236328125, -0.74658203125, 0.7080078125, -1.849609375, 0.1351318359375, 0.03668212890625, 0.04583740234375, -1.3291015625, 0.467529296875, 0.46533203125, -0.10186767578125, -0.1751708984375, -0.43505859375, -0.92724609375, 0.318359375, 0.6875, -1.1923828125, 2.162109375, -0.2030029296875, -0.9990234375, -0.712890625, -0.05682373046875, -0.62548828125, 0.296875, -1.37890625, -0.595703125, 0.30126953125, 0.478271484375, -2.025390625, 1.1865234375, 0.87451171875, -0.099609375, 1.7646484375, 0.0242919921875, -1.29296875, -0.2049560546875, -0.88720703125, 1.892578125, 0.2059326171875, 0.52783203125, 0.1400146484375, -2.109375, -1.2626953125, 0.8779296875, 1.0400390625, -0.12017822265625, -0.0994873046875, 0.49267578125, -0.318115234375, -0.8447265625, 1.6552734375, 0.62841796875, -1.453125, 0.290771484375, -1.080078125, 0.146240234375, 0.481201171875, -0.0290069580078125, 1.7421875, -0.9609375, 1.212890625, -2.125, 0.5771484375, -0.2252197265625, 1.4345703125, 2.3671875, -0.740234375, -0.152099609375, -0.440673828125, 0.1732177734375, 1.21875, -1.1279296875, 0.46875, 1.001953125, -0.0112762451171875, 1.23828125, -0.297119140625, -0.04876708984375, 0.4794921875, -1.4345703125, 1.2373046875, -0.353759765625, -0.1552734375, 0.900390625, 0.391357421875, -1.29296875, 1.150390625, 0.47509765625, 1.5244140625, -1.4599609375, 0.1810302734375, 0.67041015625, -0.06982421875, 0.309814453125, -0.92919921875, 0.45849609375, -0.79736328125, -0.02239990234375, -0.1748046875, 0.89501953125, 2.158203125, -0.11724853515625, 0.025665283203125, 0.59814453125, 1.3251953125, -0.1575927734375, 0.6083984375, -0.78857421875, 1.40625, -0.83154296875, 1.9677734375, -0.18505859375, -1.044921875, 0.3583984375, -0.95068359375, -1.5087890625, -0.134033203125, -0.0313720703125, 1.0107421875, 1.53515625, -0.9697265625, 0.91015625, -1.61328125, -0.876953125, -1.0009765625, -1.01953125, 0.26611328125, -0.337890625, 1.4814453125, -0.1923828125, -1.6494140625, 1.5009765625, 0.5810546875, 0.5361328125, -1.51953125, -0.18017578125, -0.203125, -1.048828125, 0.83203125, -0.53173828125, -0.7109375, -0.6708984375, -1.25, 0.5087890625, 1.111328125, -2.0234375, -1.03125, -0.57177734375, -1.37109375, -0.2340087890625, 1.8173828125, 0.9833984375, -0.436767578125, 0.047821044921875, -0.29736328125, -1.1484375, 1.1845703125, 0.6787109375, 0.072998046875, 0.1256103515625, -1.154296875, -1.0791015625, 0.74169921875, 0.66162109375, -1.1591796875, 0.83251953125, 0.3251953125, -1.98046875, 0.358154296875, 0.978515625, -0.26611328125, -0.356201171875, -0.138916015625, 0.092041015625, 0.3525390625, -0.4140625, 0.5478515625, 0.89794921875, 0.4609375, -1.6845703125, -0.0235595703125, 0.9345703125, 0.8173828125, 0.9765625, 1.6025390625, 0.08740234375, -0.376220703125, 1.4794921875, -1.4189453125, -0.4404296875, -0.9853515625, 0.88818359375, -0.8544921875, 0.4619140625, 1.642578125, 1.9111328125, -0.06298828125, 0.1673583984375, -0.7041015625, -0.054931640625, 1.212890625, 0.94775390625, -0.0244903564453125, 0.018707275390625, 0.75732421875, -0.875, 0.303955078125, -0.37255859375, -1.4716796875, 1.357421875, 0.158447265625, -0.6943359375, -1.25, 0.0758056640625, 0.54296875, -0.0019817352294921875, 0.045196533203125, 0.20361328125, 1.890625, -0.8896484375, 1.279296875, 0.14453125, 0.398681640625, 0.2064208984375, -0.75927734375, -1.0166015625, 0.327392578125, -2.12109375, -1.0498046875, 0.424072265625, -0.5185546875, -0.40283203125, -0.9912109375, -0.1392822265625, -0.71630859375, -1.314453125, 0.48486328125, 0.478271484375, -0.7529296875, -0.56787109375, -2.474609375, -0.57568359375, 0.71533203125, -1.671875, -0.4814453125, -0.320556640625, -0.10003662109375, 0.46826171875, -1.2958984375, 0.3759765625, 0.24853515625, -0.828125, 0.392333984375, -0.54638671875, 0.2274169921875, 0.048614501953125, -0.7509765625, 1.3330078125, 1.5009765625, -1.421875, -1.482421875, -0.84814453125, -0.48974609375, 1.9658203125, -0.1484375, 0.52587890625, 0.91357421875, -0.157958984375, 0.77294921875, -0.30517578125, 0.315185546875, -1.3740234375, 1.744140625, 1.00390625, -0.117919921875, 0.779296875, -0.2900390625, -0.9306640625, -1.2646484375, 1.4189453125, 0.6875, -0.368896484375, 0.34033203125, 0.432861328125, -1.2841796875, -0.7568359375, -0.306640625, 1.1298828125, -1.04296875, 0.31689453125, 0.2181396484375, 1.322265625, -1.3466796875, -3.08203125, 0.650390625, -1.685546875, -0.01617431640625, 1.236328125, -0.61474609375, -0.24658203125, 0.92626953125, -1.4599609375, 0.60888671875, -0.521484375, -1.0537109375, 1.3671875, 0.294189453125, -0.31201171875, -0.018402099609375, 0.27783203125, 1.03515625, 0.92578125, 1.08203125, 0.281494140625, -0.517578125, 0.448974609375, -0.6640625, -0.53662109375, -1.1767578125, -1.4462890625, 0.10772705078125, 0.2027587890625, 1.115234375, -0.117431640625, 0.1842041015625, -0.51904296875, 0.0174713134765625, -0.481689453125, 0.166015625, -1.48046875, -2.7421875, -1.169921875, 0.96484375, 0.8505859375, -0.64306640625, 1.30859375, -0.37890625, 0.0869140625, -0.422607421875, 0.0011949539184570312, -0.436279296875, -0.3125, 1.2490234375, 0.35302734375, -1.3232421875, 0.57421875, 1.3828125, 0.451416015625, -1.228515625, 0.2293701171875, -0.75, 0.53564453125, 0.48291015625, -1.779296875, -0.8310546875, -0.939453125, 0.12841796875, 0.7890625, -0.607421875, -0.39306640625, 1.84375, 1.953125, -0.60986328125, 1.0986328125, -1.546875, -0.658203125, -0.7802734375, 0.253662109375, 0.34423828125, -0.55615234375, 2.9765625, 0.8955078125, 0.6689453125, 0.11187744140625, -1.287109375, 2.466796875, 0.568359375, -0.01580810546875, 0.501953125, -0.318115234375, 0.85009765625, -1.1865234375, -1.416015625, -0.82421875, 1.0, -3.42578125, 0.298828125, 1.9326171875, -0.1373291015625, -1.3447265625, -0.388671875, 0.0526123046875, 0.1868896484375, -1.19140625, 0.449462890625, 0.08111572265625, 0.2467041015625, 0.2352294921875, -1.0078125, -0.7392578125, -0.358642578125, -0.8759765625, -0.12939453125, 0.3203125, -1.1796875, 0.035308837890625, -1.02734375, -0.7783203125, -0.202392578125, 0.2476806640625, 0.68115234375, -0.37890625, -0.90771484375, 0.371337890625, -0.303466796875, 0.71240234375, 0.392333984375, 0.8134765625, -1.328125, -0.57177734375, -1.1982421875, 0.69970703125, -0.548828125, 1.833984375, -1.6396484375, 0.73046875, 1.1416015625, 1.6142578125, -0.00746917724609375, 1.57421875, 0.62841796875, -0.20263671875, -0.0018901824951171875, -1.1669921875, 0.1292724609375, 1.013671875, -0.79248046875, -1.1005859375, -0.313232421875, 0.75732421875, -0.923828125, -0.409423828125, 0.65625, -2.458984375, 0.1448974609375, -1.14453125, 1.1572265625, 1.560546875, -1.9033203125, -0.91943359375, -0.5537109375, 1.37890625, -0.4169921875, 0.0171966552734375, -0.1141357421875, 0.85498046875, -0.7666015625, -0.5791015625, 2.27734375, 0.615234375, -2.123046875, -0.2188720703125, -0.220703125, 1.57421875, -1.5361328125, -0.253173828125, 0.50634765625, -1.0419921875, -0.4326171875, -0.63916015625, 0.51171875, 0.10205078125, -0.238037109375, -0.6611328125, 0.128173828125, 0.066650390625, 0.751953125, 2.154296875, -0.144287109375, -0.297119140625, 0.06475830078125, 1.4169921875, 1.49609375, -0.213134765625, -0.269775390625, -0.833984375, 0.36962890625, -1.744140625, -0.1383056640625, -0.61376953125, -0.443603515625, 2.265625, -0.61669921875, 0.8720703125, -0.6689453125, 1.162109375, -0.7021484375, -1.380859375, -0.36083984375, -1.078125, -2.27734375, -2.650390625, -0.0914306640625, 1.0546875, -0.045867919921875, 0.358154296875, -2.375, 1.77734375, 0.01123809814453125, -1.4306640625, -0.9921875, 0.62744140625, -0.1595458984375, -0.9951171875, -0.2154541015625, -0.13818359375, 1.0703125, 1.244140625, 2.4140625, -0.85693359375, -0.1986083984375, -1.18359375, -1.6298828125, -0.89794921875, -1.7529296875, 0.48388671875, 0.6396484375, 1.423828125, 0.70556640625, -0.2371826171875, 1.0810546875, 2.7890625, -0.5732421875, -1.1201171875, 0.2398681640625, -0.84033203125, -1.8740234375, -3.193359375, -0.95849609375, -0.64599609375, -1.4052734375, -0.65234375, 0.305908203125, 0.218505859375, 0.1680908203125, 1.60546875, 0.75244140625, -0.289794921875, 0.93994140625, 0.85595703125, 1.970703125, 1.6015625, 0.7451171875, 1.1435546875, -0.361572265625, 0.71875, -0.7939453125, -1.2490234375, 1.87890625, -0.72119140625, -0.082275390625, -2.1640625, -1.17578125, 1.1669921875, 0.69677734375, 1.0029296875, 0.72998046875, -0.454833984375, 0.05157470703125, 1.3671875, -0.2298583984375, -0.94091796875, -1.142578125, -0.51904296875, 1.8818359375, 1.005859375, -0.396240234375, -1.462890625, 0.09515380859375, -0.2432861328125, -0.89599609375, -0.16845703125, -0.0011701583862304688, -0.935546875, -0.032623291015625, -0.252197265625, 0.64111328125, -0.81396484375, -0.035980224609375, 0.392822265625, 0.9794921875, 1.1328125, 1.6572265625, -0.2354736328125, 0.89990234375, 0.1644287109375, -1.861328125, -0.08685302734375, -0.85302734375, 0.31787109375, 1.03515625, -0.005863189697265625, 0.493408203125, 1.49609375, -1.7685546875, -0.54931640625, 1.5458984375, 0.8544921875, -0.053436279296875, -1.05078125, 0.0782470703125, 1.4072265625, -0.458251953125, -0.1512451171875, -0.409423828125, 0.267822265625, 0.10699462890625, 0.379150390625, -2.369140625, 0.8935546875, -0.69580078125, -0.314453125, -1.90625, 0.64306640625, 2.21875, -0.2054443359375, 0.288330078125, -0.73095703125, -0.66650390625, 0.92626953125, -0.71142578125, -0.22314453125, 0.353271484375, -1.32421875, 0.2177734375, -0.55517578125, -0.04864501953125, -1.5126953125, -1.1708984375, -0.86328125, -1.6103515625, -0.49951171875, 0.87255859375, -0.490966796875, 0.080322265625, -0.89453125, 0.332763671875, -0.810546875, 0.93212890625, -1.99609375, -0.3173828125, 1.615234375, 1.66015625, 0.8515625, -1.8662109375, 0.69189453125, 1.1953125, -0.53857421875, 0.467041015625, 0.1461181640625, -0.415771484375, 1.587890625, 0.152099609375, 0.306396484375, 1.330078125, 1.359375, -2.58203125, 0.46337890625, 0.74169921875, -0.73388671875, 0.054229736328125, 0.7880859375, 0.8212890625, -0.64990234375, -1.2998046875, -0.68505859375, -1.115234375, 0.39013671875, 0.3642578125, 0.546875, 0.8291015625, 1.361328125, -1.5673828125, -0.439208984375, 0.174072265625, -0.2734375, 0.42919921875, 1.1328125, -0.7861328125, -0.2783203125, -0.54296875, -0.204345703125, -0.53271484375, -1.1982421875, 0.49072265625, 0.79296875, 0.89111328125, 0.88330078125, 0.5048828125, 0.0416259765625, 0.93212890625, 0.1644287109375, 1.873046875, -0.401123046875, -1.9111328125, 0.1539306640625, 0.0297088623046875, -0.341796875, 0.6025390625, 0.061279296875, -0.00836944580078125, -0.272216796875, 0.1346435546875, -1.1806640625, -0.7236328125, -0.67529296875, -0.0205078125, 0.49169921875, -0.478759765625, 0.456298828125, 0.334716796875, 0.92041015625, -1.0927734375, -0.6337890625, -0.08917236328125, -0.51904296875, -0.2486572265625, -1.181640625, -0.08837890625, 0.8212890625, -0.4794921875, -1.2275390625, 1.384765625, -2.126953125, -0.052459716796875, -1.8125, -0.46240234375, 0.00516510009765625, 1.611328125, -0.31005859375, -1.6669921875, 1.2001953125, 1.5859375, -0.75927734375, 0.77880859375, -1.083984375, -0.77783203125, 0.3642578125, -1.1494140625, -0.460205078125, 0.1353759765625, -1.0048828125, 1.5390625, -2.3671875, 1.029296875, 1.6923828125, 2.27734375, 1.3857421875, -0.68896484375, -1.1025390625, 1.0791015625, -0.35498046875, -0.40380859375, -1.7236328125, 0.17236328125, 1.6376953125, 1.0234375, 0.7939453125, -0.57275390625, 0.256591796875, -0.5830078125, 0.144775390625, -0.67578125, 0.07720947265625, 1.287109375, -0.2413330078125, 0.43994140625, -0.06494140625, 0.1866455078125, 1.6875, -0.73681640625, 0.267822265625, -0.73193359375, -0.8251953125, -0.365478515625, -1.365234375, -0.92138671875, 0.273193359375, -0.07177734375, 0.319091796875, -0.9443359375, 1.1943359375, 0.445068359375, -0.302734375, 0.3173828125, 1.037109375, -0.2919921875, 0.759765625, 0.77490234375, 0.2159423828125, 1.6279296875, -2.099609375, -0.069091796875, -0.72802734375, -1.3408203125, 0.1541748046875, -1.0126953125, -3.1328125, -1.458984375, 0.74072265625, -0.68505859375, 0.027008056640625, 1.2763671875, 0.180908203125, -0.7041015625, -1.201171875, -0.2322998046875, -0.841796875, 1.8125, -0.18896484375, -0.64208984375, 1.4521484375, 2.0234375, -1.3330078125, -0.1689453125, -0.277099609375, -0.716796875, -0.12054443359375, -0.068359375, 0.337890625, 0.65625, -0.5009765625, -0.256591796875, 0.78173828125, 0.64599609375, -0.451416015625, 1.228515625, -0.452392578125, 1.328125, -0.57568359375, 0.0477294921875, -2.201171875, -1.3359375, -0.2242431640625, 0.958984375, -0.32958984375, -1.0458984375, -0.3291015625, -1.080078125, -0.5615234375, -0.7412109375, -0.037078857421875, -0.84326171875, 0.299560546875, -0.4580078125, -2.654296875, -1.1953125, -0.921875, -0.97607421875, -1.32421875, 0.5361328125, -1.103515625, 0.1295166015625, 0.123046875, -0.7109375, -0.65625, -0.09405517578125, 0.425537109375, 0.994140625, 1.142578125, -1.0224609375, -0.058349609375, 2.046875, -1.970703125, 1.453125, -1.01171875, -1.12890625, 1.271484375, -0.273193359375, -1.154296875, -1.24609375, -0.89306640625, -1.5009765625, 0.31103515625, -0.0090789794921875, -1.3310546875, 1.55078125, -0.9765625, -0.51318359375, 0.13232421875, 0.5986328125, 0.0870361328125, -1.0517578125, 0.5537109375, 0.9755859375, -1.9970703125, -1.5654296875, -0.6337890625, -0.0867919921875, -0.63525390625, 1.1201171875, 1.30078125, 0.4423828125, 0.230712890625, -0.662109375, 1.513671875, -0.51220703125, 1.4951171875, 0.73291015625, -1.162109375, -1.4365234375, 0.77880859375, -0.54296875, -1.015625, 0.3046875, 1.6279296875, 1.255859375, -3.572265625, -0.003131866455078125, -0.97021484375, -0.55517578125, 0.1143798828125, -1.453125, 0.395751953125, -0.2137451171875, -0.70361328125, 0.303466796875, -0.81396484375, 0.1387939453125, 1.1884765625, -0.413818359375, 0.52294921875, 0.327392578125, -0.1802978515625, 0.95361328125, 0.4814453125, -0.5693359375, -0.8359375, 0.1085205078125, 0.36572265625, 0.330322265625, 1.2109375, 0.3251953125, -0.32470703125, 0.378662109375, 1.8994140625, -1.1953125, -0.52392578125, 0.420654296875, 0.288818359375, -0.302978515625, -0.410400390625, 0.37548828125, 0.265869140625, -1.2265625, -0.27490234375, 1.3115234375, 0.70703125, -0.06256103515625, 0.7900390625, -0.927734375, 0.599609375, -0.332275390625, 0.71923828125, -0.156005859375, -0.1739501953125, 0.6044921875, 0.73828125, -0.07989501953125, -0.97021484375, 1.8623046875, 1.21484375, 0.28564453125, -1.080078125, 1.6650390625, 0.2320556640625, 0.13330078125, -0.818359375, -0.92626953125, -0.037933349609375, -0.421630859375, -0.55126953125, 1.755859375, 1.4365234375, 0.6025390625, 0.88330078125, 0.7900390625, 1.0078125, -0.7861328125, 0.00862884521484375, -0.389892578125, -0.3173828125, -0.359130859375, -0.435546875, -1.103515625, 0.9638671875, 0.428955078125, 0.5498046875, -0.03778076171875, 0.387451171875, 0.94677734375, -0.849609375, -1.3486328125, 0.65087890625, 0.01525115966796875, -0.5537109375, -0.9541015625, 0.61181640625, -0.64013671875, -0.41357421875, -0.88525390625, 0.431884765625, -0.8603515625, 0.23046875, -1.0859375, -0.033447265625, 1.1005859375, -0.285888671875, -0.34375, 0.98779296875, 0.56396484375, 0.9580078125, 0.40771484375, -0.1395263671875, -0.37744140625, -0.0662841796875, -0.62158203125, -1.4228515625, -0.673828125, -1.8427734375, 0.10919189453125, 0.94970703125, 0.4208984375, -0.1431884765625, -0.8369140625, 1.12890625, -0.1676025390625, 0.7060546875, -1.17578125, -0.42529296875, -0.34619140625, 1.068359375, -1.248046875, -0.06689453125, 0.80859375, 1.8212890625, 0.71044921875, 0.75537109375, 1.2060546875, -1.154296875, -0.794921875, 0.05731201171875, 0.11724853515625, 0.95068359375, -1.013671875, -1.173828125, -1.580078125, 0.292236328125, 1.1171875, -0.364501953125, 0.368896484375, 0.66943359375, -1.3134765625, -0.86328125, 0.0733642578125, 1.3515625, 0.68359375, -1.28515625, -0.767578125, 0.010528564453125, 2.25, 0.77001953125, -0.06390380859375, -2.9375, 0.587890625, 0.96240234375, -0.8583984375, -1.48828125, 0.94091796875, 1.171875, -0.91357421875, 0.708984375, 2.09375, 3.265625, 0.366455078125, -0.72119140625, 0.02728271484375, -0.21533203125, -0.1939697265625, -0.28662109375, 0.168701171875, -1.0859375, 1.978515625, 0.60693359375, 1.1279296875, 0.798828125, 0.406982421875, 2.046875, 0.58642578125, 1.87109375, -1.25390625, -0.72412109375, 1.369140625, 0.1466064453125, 1.2734375, -0.2939453125, 0.34521484375, 1.04296875, 0.3603515625, 0.56689453125, 0.1539306640625, 0.348876953125, -0.293212890625, 1.4580078125, 0.09405517578125, 0.04815673828125, 0.398193359375, -0.152587890625, -0.254638671875, 0.0247802734375, -0.5146484375, 2.015625, -0.454833984375, -0.494873046875, -1.6318359375, -0.853515625, 0.1883544921875, -1.07421875, -1.8662109375, 0.451904296875, -0.53076171875, -1.05078125, 1.5966796875, -1.4931640625, -1.69140625, -1.181640625, -0.2408447265625, -0.0015935897827148438, 0.2373046875, -0.59326171875, 0.75732421875, -1.388671875, 1.8056640625, 0.72021484375, -0.1595458984375, 1.1884765625, 0.9677734375, 1.470703125, -0.80908203125, 1.9013671875, -0.3154296875, 1.4267578125, -0.154052734375, 0.49462890625, 0.421142578125, 0.21826171875, -0.05731201171875, -1.2705078125, 1.0810546875, 0.391845703125, -1.0888671875, 0.0217132568359375, 0.57568359375, 0.423583984375, -1.4853515625, 0.5390625, -0.8681640625, -3.041015625, 0.11126708984375, 2.576171875, 0.19384765625, 0.11273193359375, 0.27294921875, 2.115234375, 0.00881195068359375, -0.890625, 0.5859375, 0.8974609375, 0.33642578125, -0.176025390625, -1.9306640625, -0.67822265625, -0.91796875, 1.349609375, 2.7890625, 1.0068359375, -0.59765625, 0.2470703125, -0.9541015625, -0.685546875, 0.54345703125, -0.00960540771484375, -1.05078125, 1.6923828125, 0.376220703125, 0.89794921875, 0.6494140625, -0.6025390625, -0.9814453125, 1.9111328125, 1.681640625, 0.355224609375, 0.87353515625, -0.244873046875, -1.087890625, -0.97021484375, -0.0672607421875, 0.384521484375, -0.253662109375, 0.61328125, -1.4453125, -0.2476806640625, -0.05401611328125, 0.6572265625, -1.1044921875, -2.123046875, 1.173828125, -0.259033203125, -0.91552734375, 1.4619140625, -1.8974609375, 0.00705718994140625, 0.2646484375, 0.1243896484375, 0.9052734375, -1.666015625, -1.5400390625, -0.11724853515625, 0.1409912109375, -1.5830078125, -0.363037109375, 0.364990234375, -0.480224609375, 0.130126953125, 2.203125, 1.138671875, -1.5595703125, 0.8232421875, -0.322998046875, 0.95166015625, -0.701171875, -0.0242156982421875, 0.474365234375, -1.3994140625, -1.2177734375, 0.4384765625, 0.046966552734375, -1.1220703125, 1.615234375, 0.42236328125, 1.0390625, 0.29931640625, 1.361328125, -1.1787109375, -1.248046875, -1.58203125, 1.025390625, -0.364501953125, -2.205078125, -1.4189453125, 0.705078125, 0.2010498046875, -0.62548828125, 1.099609375, 0.404296875, 0.4296875, 1.087890625, -1.0068359375, 0.006420135498046875, -1.50390625, -1.0341796875, -0.45654296875, 1.3203125, 2.513671875, -0.60546875, 0.392822265625, 0.0238494873046875, 1.5322265625, 0.52685546875, -0.2313232421875, -0.09161376953125, 0.12548828125, 1.3779296875, 0.83935546875, -0.70166015625, 0.290771484375, -0.270263671875, 0.60595703125, -2.134765625, -0.488037109375, -0.272705078125, -2.701171875, 0.28369140625, 1.29296875, -0.5947265625, 1.634765625, -0.312255859375, 0.87060546875, 1.7587890625, 0.0140228271484375, -0.97412109375, -0.2069091796875, 0.82861328125, 0.2059326171875, -0.1595458984375, 0.60595703125, 1.103515625, 1.0595703125, 0.775390625, -0.2998046875, -1.0947265625, -0.45361328125, -0.66943359375, 0.35693359375, 2.072265625, -0.66943359375, -1.7763671875, 0.366943359375, -0.1754150390625, -1.1005859375, -1.2421875, 0.1275634765625, 0.8955078125, 1.2099609375, -2.271484375, 0.406982421875, -0.2181396484375, 1.3115234375, -1.8154296875, -0.81640625, -1.7890625, -1.1923828125, -1.2568359375, 0.5947265625, 0.36376953125, -0.475830078125, -0.4287109375, -0.65771484375, 1.7255859375, -0.1304931640625, -0.65869140625, -0.282958984375, -0.59130859375, 1.6591796875, -0.1302490234375, 0.89208984375, -0.156494140625, 2.38671875, 1.4345703125, 1.9482421875, 0.30517578125, -0.94873046875, -0.76708984375, 0.5185546875, 0.281982421875, -1.029296875, 0.35302734375, -1.03515625, 1.3544921875, -0.58056640625, 0.7958984375, 1.072265625, -0.303955078125, -0.95263671875, 1.5849609375, -0.1265869140625, -1.00390625, -1.365234375, -0.475341796875, 0.46044921875, 0.1717529296875, -0.392822265625, -1.158203125, -0.64306640625, -1.2119140625, -0.2237548828125, -1.591796875, -0.798828125, -0.75439453125, -0.521484375, 0.8115234375, 1.4111328125, -0.1956787109375, -0.64404296875, -0.11865234375, -1.5322265625, 1.0205078125, 0.27197265625, -0.10528564453125, -0.693359375, -0.90771484375, -0.29541015625, 0.0721435546875, -2.578125, -0.296142578125, 0.0892333984375, -1.298828125, -0.09039306640625, -1.0966796875, -0.8046875, -0.26220703125, 0.88916015625, 0.5712890625, -0.80908203125, -2.275390625, -0.2191162109375, 0.1058349609375, -0.9677734375, -0.020477294921875, 1.5263671875, 0.740234375, 0.2430419921875, -0.363525390625, 0.8505859375, -3.87109375, -1.318359375, 1.189453125, 0.66845703125, 0.335205078125, 0.4404296875, 0.2470703125, -0.1033935546875, -1.7919921875, -1.23828125, -0.87109375, 0.73046875, -0.1591796875, 0.047607421875, -1.982421875, -0.4306640625, -0.436279296875, 0.64599609375, -1.19140625, -1.091796875, -1.072265625, 1.08984375, 1.4228515625, -0.03985595703125, -1.564453125, 0.38330078125, -0.5478515625, 1.2626953125, -0.73486328125, 0.4951171875, -0.362060546875, 0.2607421875, -0.82861328125, -0.1546630859375, 0.261474609375, -0.68017578125, 0.57861328125, 0.57275390625, -0.339111328125, 0.401123046875, -0.037322998046875, -0.60986328125, 0.5673828125, -1.7236328125, 0.50634765625, 2.12109375, 1.0322265625, -1.33984375, 2.169921875, 0.5654296875, 0.66162109375, 0.363525390625, 0.364990234375, 0.400390625, 0.045623779296875, -1.2548828125, -0.465087890625, -0.685546875, -0.59619140625, -1.22265625, 1.3486328125, 1.048828125, -0.75, 1.970703125, 0.6337890625, 0.057525634765625, 1.2978515625, 0.69140625, 0.51611328125, 2.814453125, 0.185302734375, -0.66162109375, 0.56640625, -0.85888671875, -0.572265625, 1.5947265625, 2.849609375, 0.041656494140625, -0.80859375, -1.0810546875, -0.1512451171875, 0.88916015625, 0.92041015625, -0.07611083984375, 0.88232421875, -0.282470703125, -1.4716796875, -1.7080078125, 0.4541015625, 0.1773681640625, -1.228515625, -1.134765625, -0.44091796875, -1.2626953125, 0.0207977294921875, -0.443115234375, 0.83935546875, 1.0283203125, -0.57373046875, -2.1328125, -1.8759765625, -1.2265625, 1.212890625, -0.697265625, -1.076171875, 1.298828125, 1.125, 0.3515625, -1.8916015625, 0.280517578125, -0.6455078125, -1.6796875, 1.1455078125, 0.96923828125, -0.11029052734375, -0.86669921875, -3.279296875, 0.38427734375, 1.21875, 1.5810546875, 0.63671875, -0.462646484375, 1.021484375, -0.2156982421875, -0.6298828125, 0.295654296875, 0.9091796875, -2.630859375, -0.38623046875, 0.062164306640625, 1.4541015625, 1.0341796875, -1.3251953125, -0.8193359375, -0.8408203125, -0.67333984375, 0.177978515625, 1.455078125, 2.271484375, 1.658203125, -0.7021484375, -0.08197021484375, -0.107421875, 0.06597900390625, -1.337890625, 0.3671875, -0.8759765625, 1.7646484375, -1.3828125, 1.1083984375, 0.1300048828125, -2.21875, 1.1728515625, -0.002552032470703125, 1.052734375, -1.7685546875, 1.3837890625, -0.62939453125, -1.4296875, 1.1123046875, 0.88916015625, 0.599609375, 1.0283203125, 0.734375, -0.794921875, 0.3916015625, 1.193359375, -1.107421875, -0.3115234375, -1.5224609375, -1.2275390625, -0.5146484375, 1.150390625, 0.58740234375, -0.27880859375, -0.275634765625, 0.3935546875, 0.267333984375, -1.541015625, -0.66015625, 0.00266265869140625, -0.01247406005859375, 1.0166015625, -0.07391357421875, -0.97216796875, 0.83056640625, -0.9677734375, 0.0758056640625, -0.0237884521484375, -0.73974609375, -0.01983642578125, 0.658203125, 0.0372314453125, 1.1787109375, 2.0546875, 0.415283203125, -0.07830810546875, 0.09283447265625, -0.141845703125, 0.19970703125, 1.158203125, -0.96923828125, 1.2880859375, -0.11083984375, 0.2388916015625, -1.7724609375, 0.26025390625, -0.88134765625, -0.744140625, -0.04718017578125, -0.01629638671875, 1.173828125, 0.176025390625, -0.05230712890625, -1.115234375, -1.38671875, -0.3037109375, -0.481689453125, -1.1982421875, -1.455078125, 0.44091796875, 0.273193359375, -1.8271484375, -0.8837890625, -0.171630859375, -0.6865234375, -0.454345703125, 0.36328125, -0.890625, -0.625, 0.9326171875, 0.22705078125, 1.7568359375, -0.51513671875, -0.1021728515625, 0.08172607421875, -0.311767578125, 1.849609375, -0.09674072265625, -1.7890625, 0.52001953125, -0.64892578125, 0.755859375, -0.425048828125, 0.52099609375, 0.53759765625, 1.5966796875, 0.08740234375, -0.060089111328125, -1.1845703125, -0.407470703125, 1.2314453125, -0.7119140625, 1.458984375, -0.53271484375, -0.406982421875, -1.705078125, -0.939453125, -1.861328125, 0.79931640625, -0.58056640625, 1.8193359375, -0.30615234375, 1.6240234375, -0.477294921875, 0.11273193359375, 0.394775390625, -1.5068359375, -0.419677734375, 0.01195526123046875, -1.2490234375, -2.337890625, -1.1142578125, -0.4931640625, -1.0771484375, -0.7578125, -0.30419921875, -1.2431640625, 0.7548828125, 0.1929931640625, 0.09576416015625, 0.76513671875, 2.00390625, 0.90966796875, 0.400146484375, 0.183837890625, -0.6015625, -1.064453125, -0.58544921875, -0.465576171875, -1.0595703125, 0.481201171875, -1.2646484375, 1.4580078125, -0.515625, -0.83642578125, -0.30810546875, -0.95751953125, 0.95068359375, 0.29736328125, 0.8388671875, -0.7822265625, -1.7763671875, 0.276123046875, -0.50830078125, -0.348388671875, 0.1590576171875, 1.7197265625, -0.55810546875, 1.740234375, -1.6220703125, -0.276123046875, 1.767578125, -0.89990234375, -0.30078125, -1.080078125, 0.68310546875, -0.76123046875, 0.004131317138671875, -0.12408447265625, -0.192626953125, -1.5986328125, -0.353759765625, 0.08331298828125, 0.135986328125, -1.2705078125, -1.216796875, -1.9365234375, -0.658203125, 1.7646484375, 1.3115234375, -0.23486328125, -0.0233306884765625, -1.7041015625, 0.344970703125, -0.60595703125, -0.93212890625, 0.25048828125, -0.794921875, -1.041015625, -1.4912109375, 0.45458984375, -0.420654296875, -0.65625, -0.2052001953125, -1.08203125, 0.06610107421875, -0.2841796875, -0.053070068359375, 2.69921875, 0.0885009765625, 1.3740234375, 0.042938232421875, -0.86474609375, 0.46240234375, -0.03228759765625, 0.340087890625, -1.021484375, -0.2374267578125, 0.07598876953125, -0.2103271484375, 0.9541015625, -0.5576171875, 1.0732421875, -0.46630859375, 0.187744140625, -1.873046875, -1.0810546875, -1.1015625, 0.58447265625, -0.1297607421875, 0.67578125, -0.61865234375, -0.93798828125, -0.10772705078125, -0.283935546875, 0.5126953125, -0.52734375, -0.05029296875, 0.6650390625, -1.1015625, 2.203125, 0.38916015625, -1.419921875, 0.262451171875, -0.6025390625, -0.8603515625, 0.371337890625, -0.7529296875, 1.1650390625, 1.587890625, 0.37451171875, 0.052825927734375, -0.033355712890625, -0.06365966796875, -0.0058746337890625, 0.2406005859375, 0.209228515625, -0.7080078125, 0.63623046875, -1.134765625, -0.72265625, -0.2484130859375, -0.1470947265625, 0.223388671875, -1.2666015625, -0.5009765625, 0.87939453125, 0.377197265625, -0.5361328125, -0.65380859375, -1.2470703125, 1.91796875, -0.1304931640625, 0.8935546875, 0.5634765625, -0.1258544921875, 0.6279296875, -0.08502197265625, -0.260009765625, 1.525390625, 0.39306640625, 1.2724609375, -0.794921875, -0.79638671875, -0.890625, -1.6279296875, -0.93798828125, 0.10137939453125, 0.0120849609375, -0.311279296875, -0.09271240234375, 2.171875, 0.261962890625, -1.1982421875, 0.546875, -0.447509765625, 0.18701171875, 0.33056640625, -0.9775390625, -0.47998046875, -0.023345947265625, 0.2080078125, 1.0458984375, 0.98291015625, -0.09814453125, 0.572265625, -0.94384765625, -0.99267578125, 1.5029296875, -1.478515625, 0.19921875, -0.96826171875, -0.7265625, -0.89013671875, -0.2471923828125, 0.84619140625, -0.8408203125, 1.3525390625, -0.63427734375, -0.7587890625, 0.7900390625, 0.7998046875, 0.393798828125, -1.4736328125, 0.99755859375, 0.3125, -0.398681640625, 0.4853515625, 1.6083984375, -1.0546875, -0.36474609375, -0.7080078125, 1.09375, 0.307373046875, 0.448486328125, 0.438720703125, 1.6708984375, 1.720703125, 0.46044921875, 0.6611328125, -0.426513671875, 0.12225341796875, 0.9423828125, -0.568359375, 1.197265625, 0.75390625, -0.55859375, 0.50439453125, 0.95947265625, 0.96826171875, -0.31591796875, -0.074462890625, 1.154296875, 0.64697265625, 0.92138671875, -0.94091796875, -2.65234375, -1.017578125, 2.408203125, -0.1893310546875, -0.775390625, -1.2509765625, -0.7099609375, 0.56298828125, 0.57763671875, 0.8701171875, 0.43212890625, -0.495849609375, -0.418212890625, -1.12109375, 1.5869140625, 0.062042236328125, 0.52783203125, 0.93310546875, -1.421875, 1.2314453125, -0.01885986328125, 0.599609375, 2.033203125, -0.669921875, -1.3779296875, -0.54736328125, -1.8505859375, 0.35400390625, -0.1361083984375, 2.09375, -0.5537109375, -0.35693359375, 0.02581787109375, 1.564453125, -0.79150390625, -1.232421875, -0.31103515625, -2.583984375, 1.81640625, 0.6650390625, -1.25, 0.36474609375, -0.218505859375, 0.46728515625, -0.88818359375, -0.019561767578125, 1.4375, 1.5546875, -0.2203369140625, -0.2890625, -1.23828125, -0.7021484375, -0.52587890625, -0.0230255126953125, 0.4404296875, -0.27099609375, 1.81640625, 0.303955078125, 0.4326171875, -1.275390625, 1.2021484375, 1.716796875, -1.0439453125, -2.580078125, -1.314453125, 0.291259765625, -1.30859375, 0.33642578125, 0.264404296875, 0.552734375, -0.6240234375, 1.748046875, 0.146240234375, -0.2802734375, -0.7197265625, 0.73095703125, 0.436279296875, 0.280029296875, 1.3408203125, -1.283203125, 0.9560546875, -1.119140625, -0.5, -1.5712890625, -1.51953125, -0.48193359375, 0.36572265625, 0.6767578125, 0.6796875, -0.387939453125, 0.3251953125, 1.2822265625, -1.2822265625, 1.3974609375, -1.7236328125, 0.2880859375, -1.6806640625, -0.122802734375, -0.308837890625, 0.171630859375, -0.63720703125, -0.189697265625, -0.08856201171875, 1.0556640625, 0.85791015625, -0.7734375, -0.43896484375, -0.62646484375, 0.061737060546875, -1.44140625, -1.1796875, 2.361328125, 0.27734375, 0.08734130859375, -0.4267578125, -0.0181427001953125, -1.4599609375, -0.642578125, -0.7841796875, 0.322998046875, 0.9560546875, 0.104736328125, 0.74267578125, 0.370849609375, -1.615234375, 2.55859375, -0.05120849609375, -0.94482421875, 0.9599609375, 1.212890625, 0.513671875, 1.0078125, 0.24365234375, 0.84326171875, -1.4130859375, -0.069091796875, -1.5966796875, -1.3564453125, -0.494140625, 1.0556640625, 0.176513671875, 0.6025390625, 0.58740234375, 1.4326171875, -0.0914306640625, -1.12890625, -1.17578125, 0.425048828125, 1.0791015625, -1.5029296875, -1.2255859375, -2.53515625, 1.8857421875, 0.0572509765625, -0.08935546875, 1.3623046875, 0.38623046875, -0.61767578125, 0.51513671875, -0.1768798828125, -0.1756591796875, 0.477783203125, 0.31005859375, -0.533203125, 1.33984375, 1.6123046875, 1.3447265625, -1.4365234375, 0.39111328125, -1.5146484375, 0.73388671875, -0.076904296875, -0.5634765625, 0.3466796875, -0.2060546875, -0.99560546875, 0.69775390625, -1.5126953125, 0.09246826171875, 0.73583984375, -0.29638671875, 0.7080078125, -0.18310546875, -0.8447265625, -1.6552734375, -0.1268310546875, 0.147705078125, 0.748046875, 0.580078125, -2.947265625, 1.0, 1.177734375, 0.00585174560546875, 0.51953125, -0.421630859375, -1.7275390625, -0.052337646484375, 0.1474609375, -0.0280609130859375, -2.595703125, 0.258056640625, -0.01045989990234375, -0.049896240234375, -0.408447265625, -0.03759765625, 0.9384765625, -1.669921875, 2.078125, -0.974609375, -0.70263671875, 2.466796875, 0.8369140625, 1.6591796875, -0.0149688720703125, -0.223388671875, -1.5615234375, 0.70947265625, -0.806640625, -0.55419921875, 2.435546875, -0.91845703125, -0.7119140625, -0.36474609375, 0.67822265625, 0.984375, -1.2802734375, 0.5634765625, -1.18359375, -0.5986328125, 0.039642333984375, 0.77294921875, -0.521484375, -1.4443359375, -0.25927734375, 0.04901123046875, -0.263916015625, 2.1171875, 0.529296875, 1.5029296875, -0.244873046875, -0.2724609375, 2.095703125, 1.49609375, -0.383544921875, 0.73291015625, -1.2421875, -0.2022705078125, 0.97998046875, 1.30078125, -0.34130859375, 0.2176513671875, 0.04693603515625, -0.105712890625, -0.75634765625, -1.517578125, -1.1279296875, -0.1427001953125, -0.349609375, -1.4404296875, -0.08380126953125, -1.640625, 0.630859375, 0.2039794921875, -0.1669921875, 0.4287109375, -0.1654052734375, 0.273681640625, -0.94580078125, -0.215087890625, -0.89111328125, 0.8271484375, -0.14501953125, 1.220703125, 0.0233001708984375, 1.37890625, 0.2880859375, 1.1181640625, -0.60595703125, -0.6416015625, 0.61376953125, 1.2412109375, 2.404296875, 0.60888671875, 0.4775390625, 1.212890625, -0.1796875, 0.019500732421875, 0.06646728515625, 1.7451171875, 1.3779296875, 0.52099609375, -0.258544921875, 0.93017578125, -0.69873046875, -0.6142578125, 0.322265625, -0.9794921875, -0.88720703125, -0.0015354156494140625, 0.74072265625, -1.1328125, 0.58447265625, -0.83544921875, -0.97412109375, 1.220703125, 0.13232421875, -0.0010557174682617188, -0.95556640625, 0.8369140625, 0.69775390625, -0.83935546875, 0.359375, -0.07586669921875, 1.8359375, 0.802734375, 0.5888671875, 0.60009765625, -0.63623046875, -0.26953125, 1.44921875, 0.1368408203125, 0.1265869140625, -0.81591796875, 0.6171875, -0.552734375, 0.2384033203125, 0.73193359375, -0.08978271484375, -0.91845703125, 1.17578125, 1.83984375, 0.04278564453125, 0.48193359375, -0.1549072265625, -0.264404296875, -0.11285400390625, -0.266845703125, -0.8095703125, -0.459228515625, -0.09039306640625, -0.03228759765625, 0.37939453125, -1.1474609375, -0.464111328125, -0.98046875, -0.343994140625, -1.5322265625, 1.970703125, -0.235107421875, -1.6494140625, 0.325439453125, 0.53125, -0.31005859375, 0.95654296875, 1.845703125, -0.51904296875, 0.46142578125, -2.080078125, -0.280517578125, 0.85400390625, -1.7822265625, 1.1025390625, -0.036376953125, -0.315185546875, 0.336669921875, -1.2900390625, -0.6162109375, 0.83056640625, 2.072265625, -1.232421875, 0.720703125, -0.287109375, 0.391357421875, -0.94091796875, 0.7001953125, -1.2177734375, 0.86865234375, 0.452392578125, 1.845703125, 1.38671875, 2.0234375, -1.091796875, 2.11328125, 0.06201171875, 0.82666015625, -1.1796875, -0.470703125, -0.105712890625, -1.4482421875, 1.3701171875, 0.078125, -0.365478515625, 1.8291015625, -0.94091796875, 0.6826171875, -0.46826171875, -1.8955078125, 0.94580078125, -0.0994873046875, -0.828125, -0.052337646484375, 0.433349609375, 0.625, -0.3994140625, -0.8408203125, 0.7412109375, -0.9208984375, 0.724609375, 0.029327392578125, -0.7734375, 0.057464599609375, 0.26513671875, 1.080078125, 2.603515625, -0.93408203125, 0.01496124267578125, 0.65087890625, -0.273193359375, -2.25390625, 1.6494140625, -1.0830078125, 1.8115234375, 1.5078125, -1.728515625, 0.1484375, 1.1884765625, 0.4248046875, 0.08941650390625, -0.78759765625, 0.218017578125, 0.64501953125, -2.5234375, 0.64208984375, -0.388427734375, -0.10699462890625, 1.8525390625, -0.630859375, -0.462158203125, -1.8759765625, -0.6845703125, -2.166015625, 1.908203125, -0.8427734375, -0.86474609375, -1.0439453125, 0.6845703125, -1.4833984375, -1.5361328125, -0.55517578125, 0.896484375, 0.96923828125, -0.61328125, -1.1396484375, -0.060699462890625, -0.408935546875, -0.158935546875, 0.3310546875, -0.409912109375, 2.173828125, 0.51220703125, 1.2861328125, 1.2734375, -1.0908203125, -1.177734375, -0.646484375, 0.362548828125, -0.51513671875, -0.2880859375, 0.36669921875, -0.0054473876953125, -1.0439453125, 0.5625, -0.845703125, -0.5390625, -0.0192108154296875, -0.6396484375, -0.51416015625, 0.27783203125, -0.58154296875, -0.433837890625, -0.751953125, 0.482177734375, -0.0904541015625, 1.3193359375, -1.8134765625, 0.5078125, -0.437744140625, -1.283203125, -1.263671875, -1.3046875, 1.7080078125, 1.1103515625, 0.1510009765625, 0.55029296875, -0.7294921875, 0.71044921875, -0.13330078125, 0.376220703125, 1.1455078125, 0.051849365234375, -0.244873046875, 0.30908203125, 1.1015625, -0.305419921875, 0.787109375, 0.95654296875, -1.302734375, -0.1527099609375, 0.48876953125, 0.407958984375, 0.176513671875, 0.404541015625, 0.122314453125, -0.306884765625, -0.0704345703125, -0.66943359375, -0.73828125, -1.0439453125, 1.6923828125, 0.61279296875, 0.1683349609375, 0.130126953125, -0.413818359375, 1.86328125, 1.5634765625, 0.68408203125, 0.69384765625, 0.2410888671875, -1.75, 0.437255859375, -0.81591796875, 1.6533203125, -0.5048828125, 0.11871337890625, -1.515625, 0.52490234375, 1.1162109375, 1.42578125, -0.9736328125, 0.1317138671875, 2.443359375, -1.01171875, 0.4140625, -0.334228515625, -0.71142578125, 1.0185546875, -0.91357421875, -0.85693359375, 0.0208740234375, -0.97265625, 0.482666015625, 1.232421875, -1.8505859375, -1.1767578125, 0.5791015625, -1.220703125, -1.3505859375, 1.8369140625, -1.1748046875, -0.91748046875, -0.6826171875, -0.301513671875, 1.8515625, 1.1416015625, 1.1611328125, 0.92333984375, -0.017059326171875, -0.5068359375, -0.9736328125, -2.58203125, 1.6435546875, 0.058990478515625, 0.6201171875, 0.32568359375, 1.01171875, -0.67578125, 0.51904296875, 0.12030029296875, -0.268310546875, 1.115234375, 0.7080078125, 0.4169921875, -0.01300048828125, 1.9912109375, -0.038177490234375, -0.0372314453125, -0.006683349609375, 0.48486328125, 0.455078125, -0.78076171875, -0.900390625, -1.294921875, -0.2435302734375, 0.5068359375, 1.4814453125, -0.364990234375, 0.112548828125, 0.4423828125, 0.0963134765625, 0.27734375, 0.728515625, -0.87939453125, -0.078857421875, 0.361572265625, -0.5107421875, -1.1455078125, -0.0931396484375, -0.446533203125, 2.58203125, 0.22509765625, 0.486083984375, 0.1419677734375, -3.16015625, 0.186767578125, -0.07183837890625, -1.6416015625, -2.166015625, -0.148193359375, -0.77783203125, 0.10650634765625, 1.7626953125, -0.11126708984375, -1.0400390625, -0.50341796875, 0.1361083984375, -1.822265625, 0.446533203125, -0.798828125, 0.908203125, 0.70849609375, -0.27294921875, 1.0126953125, -0.39990234375, -0.88427734375, 1.6162109375, -1.18359375, 0.18212890625, -0.72705078125, 1.0478515625, -0.409423828125, 0.80615234375, -0.70849609375, 1.404296875, -0.66455078125, 0.33642578125, 0.0992431640625, 0.11029052734375, -0.72314453125, -0.1910400390625, -0.83203125, -0.92236328125, 1.1923828125, 0.45849609375, 1.6357421875, 1.97265625, 0.11016845703125, 1.556640625, 1.0908203125, -0.33447265625, 0.57470703125, 1.126953125, 0.7392578125, -0.65185546875, 0.4375, -0.32470703125, 0.84912109375, 0.8271484375, -0.95556640625, 2.603515625, -1.3720703125, 0.213623046875, 0.74560546875, 0.23828125, 0.8603515625, -1.234375, 0.1527099609375, -0.57421875, 1.8837890625, -0.8662109375, -0.8740234375, 0.0667724609375, -1.052734375, 0.1331787109375, 1.7607421875, -1.3447265625, 2.791015625, 0.7900390625, 0.2413330078125, 2.40234375, 0.181396484375, 0.3623046875, -0.00878143310546875, -1.349609375, -0.367431640625, -0.82958984375, -0.22607421875, -0.28076171875, -0.8701171875, 0.55126953125, 1.0380859375, -0.400634765625, 0.441650390625, -1.0927734375, -0.286376953125, 0.472900390625, -0.63916015625, 0.85791015625, 1.359375, -0.94091796875, 0.477783203125, -0.09490966796875, 0.006839752197265625, -0.6357421875, 0.53173828125, -2.087890625, 0.239990234375, -1.681640625, 2.171875, -0.806640625, -0.278076171875, -1.275390625, 0.666015625, -0.71484375, 0.81591796875, -0.7744140625, 1.310546875, 1.9228515625, 0.233154296875, -0.462646484375, 0.220947265625, -0.023040771484375, -0.0054931640625, -0.0005803108215332031, -0.80859375, 0.55078125, 0.64697265625, 0.60791015625, -2.01171875, -0.3115234375, -0.2435302734375, -0.11077880859375, -0.183349609375, 0.21630859375, -1.1142578125, 1.4765625, 0.09710693359375, 0.09832763671875, -1.314453125, 0.66015625, 0.28564453125, -0.80908203125, 0.2308349609375, -0.126953125, 1.0615234375, -0.1490478515625, -0.41552734375, 0.47412109375, 0.437255859375, -1.24609375, -0.2242431640625, -1.12890625, -0.34228515625, -1.82421875, 0.394287109375, -1.2724609375, -0.45751953125, 0.164306640625, -2.19140625, 0.07666015625, 2.779296875, -0.31494140625, 0.41748046875, 0.98046875, 0.458984375, -0.80810546875, -0.1995849609375, -0.93359375, 1.2353515625, -0.0830078125, -1.2646484375, 0.19287109375, 1.2978515625, -0.36962890625, 0.0252838134765625, -1.658203125, -1.375, 2.509765625, -1.0732421875, -0.92626953125, -1.5625, -0.564453125, -1.41015625, -0.25439453125, -0.1802978515625, 0.08740234375, -1.484375, 1.3828125, -0.68701171875, 0.58251953125, 0.2381591796875, -0.255126953125, -0.164306640625, 0.8828125, 0.06414794921875, 1.2900390625, -0.5732421875, 0.67578125, 0.482421875, 0.391357421875, -2.115234375, 1.3671875, -0.47216796875, 0.427001953125, 0.06390380859375, 0.9248046875, 0.478271484375, -0.046783447265625, -1.4814453125, 0.26220703125, -0.443603515625, -0.56689453125, 1.2197265625, -1.0048828125, -1.435546875, -0.04541015625, 1.068359375, 2.41015625, 0.75439453125, 0.7958984375, 0.336669921875, 0.548828125, -0.578125, 0.140869140625, 2.12109375, 1.3740234375, -1.5263671875, 0.268798828125, 0.44970703125, 0.196533203125, -1.888671875, -1.0537109375, 1.0283203125, -0.6220703125, -0.30322265625, -0.79248046875, 0.892578125, 1.2646484375, -0.50341796875, 0.1458740234375, -0.058013916015625, 1.7353515625, -0.471435546875, -0.14404296875, -0.54150390625, -1.13671875, 0.7041015625, -0.9111328125, 1.427734375, -0.92724609375, 0.2578125, 1.5341796875, 1.0888671875, 1.8232421875, -1.0634765625, 1.814453125, -0.298828125, 1.3076171875, -1.111328125, 0.84521484375, -0.10101318359375, -0.66650390625, 2.248046875, -1.04296875, -0.1611328125, 0.35498046875, -0.215087890625, -0.387939453125, 0.98779296875, -1.068359375, 0.467529296875, 0.372802734375, -0.2734375, -2.25390625, 0.71435546875, -0.5830078125, -0.04681396484375, 0.52490234375, -0.48388671875, -0.246337890625, -1.107421875, -0.28759765625, -0.457275390625, -0.42236328125, -0.56689453125, 1.064453125, 0.52001953125, 1.291015625, -1.0771484375, -0.08282470703125, -1.7470703125, 0.7685546875, -0.259521484375, 0.67919921875, -0.3525390625, 0.1646728515625, 0.2498779296875, 0.10491943359375, 0.08544921875, 0.042694091796875, -0.08514404296875, 0.452880859375, -0.14013671875, 0.10101318359375, -0.6279296875, 1.2919921875, -1.41796875, -0.630859375, 1.017578125, -0.65966796875, 0.66943359375, -1.4462890625, -0.70751953125, -0.95703125, -0.88525390625, 2.69921875, 0.0149993896484375, -0.64404296875, -1.1044921875, 0.15966796875, -0.873046875, -0.219970703125, -0.94384765625, -0.82421875, -0.92236328125, 0.360107421875, -1.150390625, 0.412353515625, 1.0830078125, 0.270751953125, 1.9130859375, 0.312255859375, -2.09375, -0.259033203125, -0.2041015625, 0.12176513671875, 2.3671875, 0.77978515625, 0.58935546875, 0.53369140625, -0.9267578125, -0.67626953125, -0.90625, -0.95703125, -1.7978515625, 1.615234375, 0.09820556640625, -0.58642578125, 0.63330078125, -0.193359375, 0.49169921875, -2.015625, 0.13623046875, -0.6650390625, 0.2464599609375, 0.475830078125, 0.9296875, 1.3271484375, -1.0322265625, 1.134765625, -0.11846923828125, 0.96630859375, 0.74853515625, -1.6162109375, -0.276611328125, -0.47412109375, 0.53173828125, 0.79150390625, -0.83251953125, -0.13916015625, 0.1488037109375, -0.272216796875, 1.587890625, -0.2308349609375, -0.63916015625, 0.2425537109375, 0.82763671875, -0.41845703125, 0.29296875, 0.572265625, -0.51318359375, -0.5166015625, 0.72509765625, -0.67431640625, 1.259765625, 0.505859375, 0.67138671875, 0.256103515625, -0.066162109375, -0.384765625, -1.19140625, -0.5166015625, 0.297607421875, -0.76123046875, 0.708984375, 2.228515625, -1.65625, 0.83740234375, -1.435546875, -0.480712890625, 0.198486328125, -0.1146240234375, -0.88427734375, 0.49169921875, 0.0638427734375, 1.1572265625, 0.7109375, 0.80078125, -1.6123046875, 0.369873046875, -0.5732421875, 1.0888671875, 0.478515625, 2.57421875, 0.1102294921875, -2.12890625, -0.45947265625, 0.428955078125, 1.357421875, -0.00868988037109375, -0.369140625, -1.1162109375, 0.0843505859375, -0.56005859375, 1.671875, -0.8388671875, 0.78857421875, -1.12890625, -0.479736328125, -0.1923828125, 0.01300048828125, -0.55419921875, 0.74072265625, -0.0870361328125, 1.3740234375, -0.432373046875, -0.5771484375, -0.98095703125, -1.3623046875, 0.1617431640625, 0.72265625, -0.4228515625, 1.009765625, 2.443359375, -2.35546875, 1.8701171875, -0.7421875, 1.322265625, -1.5927734375, -0.259765625, 0.1895751953125, 0.493896484375, -0.302978515625, -0.2177734375, -0.392578125, -2.396484375, 0.361328125, 0.90869140625, -1.10546875, 0.7958984375, -1.2783203125, -1.08203125, 0.376708984375, 0.9091796875, 0.63720703125, -0.372802734375, 0.1688232421875, 1.6025390625, 0.2154541015625, 0.268310546875, -0.375, 0.93115234375, -1.646484375, -0.118408203125, 0.703125, 0.275146484375, -0.6044921875, 1.31640625, 0.042572021484375, -0.81640625, -1.755859375, -1.05859375, -1.1640625, -0.2398681640625, 0.80908203125, -1.6044921875, -1.16015625, -0.369873046875, 1.5, -0.2119140625, -0.71923828125, -0.65576171875, -0.8740234375, -1.04296875, 0.525390625, -0.2822265625, -0.3798828125, -0.2181396484375, 0.09857177734375, -0.32421875, 1.1591796875, -1.1044921875, -1.375, 0.4052734375, -1.4609375, 1.06640625, -0.040924072265625, 1.1513671875, -0.453125, 0.8916015625, 0.337158203125, -1.2392578125, -0.71923828125, -0.31103515625, 0.541015625, 0.12255859375, 0.79150390625, -1.384765625, -1.205078125, -0.00860595703125, -1.7333984375, 0.4345703125, -0.10638427734375, 0.284912109375, -1.3544921875, -1.7060546875, -0.37109375, -1.0244140625, -2.466796875, -1.623046875, -0.78271484375, -0.57861328125, 1.580078125, -0.18505859375, -1.490234375, 0.09710693359375, 0.521484375, 1.1083984375, 0.055145263671875, 0.4990234375, 1.5517578125, -0.455078125, -0.40966796875, -1.193359375, 1.392578125, 1.2900390625, -0.99755859375, -1.86328125, 2.09765625, 0.10418701171875, 1.1416015625, -0.8486328125, 0.85546875, -0.6865234375, 0.73095703125, 0.64111328125, 0.92041015625, -0.50830078125, 0.2060546875, 0.74755859375, 0.2861328125, -1.4189453125, 0.195556640625, -0.07177734375, -0.97216796875, -0.86572265625, -1.0458984375, -0.84814453125, 0.154052734375, -0.204833984375, -1.369140625, -0.39306640625, 0.65185546875, -1.48828125, 2.171875, -0.331298828125, 0.139892578125, -0.85986328125, -1.32421875, -0.609375, -0.322998046875, -0.755859375, 1.01171875, 1.6640625, -0.1561279296875, 0.205322265625, -0.148681640625, 0.87646484375, -0.88330078125, 0.358642578125, 0.1907958984375, -0.3564453125, -1.947265625, 1.2607421875, -1.9580078125, -0.202392578125, 1.1201171875, 1.9375, 1.0341796875, -0.41162109375, 0.955078125, 1.0048828125, 0.7421875, 1.5283203125, 0.35888671875, 1.3203125, 1.90625, -0.342529296875, -0.352294921875, -0.319091796875, -1.154296875, -1.2744140625, -0.671875, -0.066650390625, 2.17578125, -0.489990234375, -0.7001953125, 0.195068359375, -1.2578125, -1.140625, 2.189453125, 0.99951171875, -0.302490234375, 2.048828125, 0.461181640625, -0.01024627685546875, -0.8076171875, 1.7333984375, 0.662109375, 0.346435546875, -0.367919921875, 0.2017822265625, -1.6435546875, 0.9853515625, -0.350830078125, -2.5703125, 0.48095703125, 1.2783203125, 0.369140625, 0.70166015625, -0.167724609375, 0.5166015625, -0.81591796875, 0.251953125, -0.284423828125, -1.2470703125, -1.1259765625, 1.806640625, -0.90234375, -0.8505859375, -0.403564453125, -1.9228515625, 0.849609375, 0.110595703125, 0.478271484375, 0.66162109375, -1.9501953125, -0.052886962890625, 0.0751953125, -0.44384765625, 0.47021484375, 0.08526611328125, -0.93017578125, -0.87841796875, 0.71484375, 1.2001953125, -1.19140625, -1.9482421875, -0.80712890625, 1.90625, 0.63720703125, -1.224609375, -1.18359375, 0.255859375, 0.00909423828125, -1.9404296875, -0.43798828125, 0.1453857421875, 1.8037109375, -0.53125, -1.3310546875, 0.48876953125, -1.078125, 0.7236328125, -1.333984375, 0.01436614990234375, 0.5029296875, -0.8076171875, -0.479736328125, -0.77490234375, 0.282958984375, 0.350341796875, -0.03900146484375, -0.451171875, 0.286865234375, -0.72265625, -0.73974609375, -0.638671875, 1.4853515625, 0.1842041015625, 0.603515625, 0.09307861328125, -0.95654296875, 1.5947265625, 0.00048232078552246094, 2.01171875, 0.10504150390625, 0.07781982421875, -1.05859375, -1.7255859375, -0.17333984375, -1.0458984375, -0.0482177734375, 0.08306884765625, -0.7099609375, -0.583984375, -1.4912109375, -0.2548828125, 0.388427734375, 0.66357421875, 0.61572265625, -0.1466064453125, -0.7001953125, -0.79345703125, -0.3056640625, 1.9267578125, 1.1796875, -0.51416015625, 1.931640625, -0.0404052734375, -0.8818359375, -0.254638671875, -0.45947265625, -1.6845703125, 1.388671875, 0.1982421875, 1.8115234375, -1.810546875, 0.09954833984375, -0.8046875, 0.034088134765625, -1.5390625, 1.5390625, -0.322998046875, -0.64501953125, -1.0576171875, 1.1357421875, 0.3427734375, 0.90576171875, -1.5361328125, 1.0615234375, -0.473876953125, -0.923828125, 1.03125, 0.1412353515625, 0.54541015625, 0.2958984375, -1.3251953125, 0.08941650390625, -0.1536865234375, 2.248046875, 0.296630859375, -1.0791015625, 0.82470703125, -0.052398681640625, -0.3896484375, -0.33349609375, -1.2236328125, -1.2568359375, 0.299072265625, -0.07061767578125, -0.33935546875, 0.7294921875, -0.87890625, 0.385986328125, 2.306640625, 0.82080078125, 0.3251953125, 0.442626953125, 0.0758056640625, -0.94189453125, 0.172119140625, 0.1790771484375, 0.74462890625, -0.705078125, 1.5, -0.30322265625, -0.448486328125, 1.037109375, 0.4716796875, -0.175048828125, 0.94287109375, -0.99267578125, 0.037689208984375, 1.169921875, 1.02734375, 0.2403564453125, -0.490478515625, 0.8515625, 0.97314453125, 0.74853515625, 1.6484375, -1.2275390625, 0.93310546875, -0.07623291015625, -0.728515625, -1.7177734375, 0.98291015625, 0.8642578125, 0.2037353515625, 1.1201171875, 1.32421875, -1.2646484375, -2.064453125, 0.053863525390625, 0.282470703125, -1.46484375, 0.65380859375, -0.281982421875, 0.57861328125, -0.50537109375, -0.322998046875, 0.50927734375, -0.6240234375, 0.93505859375, 0.9794921875, -0.427978515625, -0.625, 0.0941162109375, -0.00449371337890625, 1.0458984375, 1.2646484375, -0.5986328125, -0.0016651153564453125, -1.2353515625, 0.62451171875, -0.5517578125, 0.7685546875, 1.05078125, 0.445068359375, 0.0253143310546875, -1.6259765625, -0.0625, 0.8994140625, 0.452880859375, -0.4130859375, -1.5927734375, 1.076171875, 0.84228515625, -0.31689453125, 0.48583984375, -0.556640625, 0.234619140625, 0.58544921875, 1.3505859375, 0.53759765625, 0.2398681640625, -1.814453125, 0.07098388671875, 1.0751953125, 1.7333984375, -0.402587890625, 0.9921875, -1.3095703125, -0.450439453125, 0.78466796875, -0.2208251953125, -0.10455322265625, 0.4384765625, -1.78515625, 0.53857421875, 0.1971435546875, -0.08984375, 0.81103515625, -0.039031982421875, 0.167724609375, -0.8076171875, -0.736328125, -0.5205078125, -0.740234375, -0.092041015625, 1.083984375, 1.625, -0.5302734375, 0.91064453125, 0.423828125, -0.45166015625, -0.9091796875, 0.72265625, 1.9375, 1.513671875, 0.5458984375, 0.419189453125, -0.712890625, -0.34814453125, -0.61669921875, 0.193115234375, -0.4921875, 1.0029296875, 0.0653076171875, 0.546875, 0.349853515625, 1.4072265625, 0.01959228515625, 0.90576171875, 0.306884765625, 1.033203125, 1.2197265625, 0.0606689453125, -0.4267578125, 0.732421875, 0.349365234375, -0.8671875, 1.27734375, 0.70263671875, -0.6142578125, 0.25390625, -0.37841796875, 0.09222412109375, 0.48046875, 0.33935546875, 1.193359375, 1.8525390625, 0.63818359375, 0.026947021484375, 0.30859375, 1.013671875, 1.2880859375, -0.763671875, 0.27294921875, -0.338134765625, -0.319091796875, -1.1630859375, -0.450439453125, -1.517578125, -0.81640625, 0.21533203125, -0.2086181640625, -0.60302734375, -0.29150390625, 2.5859375, -0.334228515625, -0.75048828125, 1.1630859375, 0.90576171875, 0.005733489990234375, -0.5908203125, 0.9697265625, 0.59375, -0.287841796875, -0.7626953125, -0.70654296875, 0.015350341796875, 0.76318359375, 1.4892578125, 0.1512451171875, -1.4873046875, -0.99658203125, 1.5126953125, -1.65625, -0.25537109375, -0.87451171875, -0.257080078125, -0.71240234375, 1.4375, 0.52001953125, -0.859375, -0.299560546875, -0.89208984375, 1.126953125, -1.1416015625, -0.35205078125, -0.8525390625, -1.2841796875, 0.49609375, 1.537109375, 0.420654296875, 0.1529541015625, 0.54052734375, -2.373046875, -1.46875, 0.050567626953125, -0.98046875, -0.1329345703125, -1.2763671875, -0.284423828125, 0.474609375, -0.442626953125, -1.7578125, 1.2177734375, 0.76123046875, 0.2213134765625, -0.5634765625, -0.1336669921875, 1.0615234375, -0.30517578125, -0.59228515625, -0.1317138671875, 0.1810302734375, 1.1005859375, -0.14990234375, -0.0029163360595703125, 2.087890625, 0.20361328125, -1.1435546875, 0.3330078125, 0.96484375, 0.213134765625, -1.0107421875, -0.03851318359375, 0.88671875, 0.6044921875, -0.97412109375, -2.091796875, 0.68115234375, 1.0078125, 1.115234375, -1.2158203125, 0.861328125, 1.326171875, -0.0850830078125, -0.0357666015625, 0.93505859375, -2.078125, 0.43505859375, -1.498046875, -0.2052001953125, 1.6513671875, -0.188232421875, -0.2359619140625, 0.33349609375, 0.94775390625, 1.7822265625, -0.24853515625, 0.64404296875, -1.4130859375, 0.380615234375, -0.362060546875, 1.169921875, -0.41259765625, -1.7314453125, 1.16796875, -1.8671875, -1.9892578125, -0.48486328125, 0.8037109375, 1.66015625, -0.9326171875, 1.9140625, 1.0380859375, -0.94921875, 1.287109375, -0.12744140625, 0.109130859375, -0.59130859375, 0.6953125, -0.124267578125, -2.28125, 0.361572265625, 1.1298828125, 1.12109375, -0.9794921875, 0.54150390625, 0.6533203125, 1.2978515625, -1.388671875, -0.59619140625, 0.4072265625, -0.033935546875, 1.2724609375, -3.052734375, 1.5869140625, 0.8740234375, 1.3056640625, -0.2086181640625, -0.39794921875, -0.62646484375, 1.310546875, 0.179931640625, 0.99853515625, -0.5439453125, 1.8876953125, 1.2080078125, -1.6064453125, 1.1962890625, -0.4775390625, 0.236083984375, 0.253173828125, 0.2138671875, -0.939453125, -0.049591064453125, -1.572265625, -0.427490234375, 0.2890625, -0.0626220703125, -0.27685546875, -0.1712646484375, -1.8427734375, 0.599609375, -1.2880859375, -1.095703125, 0.64453125, -0.05828857421875, 0.58837890625, 0.3603515625, -0.623046875, 1.1962890625, -0.6416015625, -0.74169921875, -0.1243896484375, 0.201416015625, -0.1673583984375, -1.1162109375, -0.474365234375, -0.630859375, -0.7861328125, -0.17041015625, -2.66796875, 1.619140625, -0.029449462890625, -1.0126953125, -0.294189453125, 0.65673828125, -0.99072265625, -0.076171875, 0.416748046875, 1.6806640625, 0.794921875, 0.10791015625, -0.1768798828125, 0.338134765625, -0.27490234375, 1.3623046875, 0.06829833984375, 0.29833984375, -0.1153564453125, 0.06005859375, 0.69775390625, 0.55615234375, 0.0709228515625, -0.11639404296875, -0.13720703125, 1.404296875, -1.37109375, 0.79248046875, 0.2841796875, 0.75927734375, 0.40625, 0.27001953125, -1.1123046875, -1.0517578125, -1.19140625, 0.91259765625, 1.318359375, 0.203125, 0.0270233154296875, -1.373046875, -2.728515625, -1.0478515625, 0.42041015625, 0.323486328125, -0.1903076171875, 0.630859375, 0.5712890625, -0.22607421875, 0.11859130859375, -2.541015625, 1.7109375, 0.15380859375, 1.7080078125, 0.05389404296875, 1.0146484375, 0.853515625, -0.857421875, -0.2235107421875, -0.61865234375, 0.51220703125, -0.305908203125, 0.951171875, -1.8642578125, 1.2958984375, 2.1015625, 0.82666015625, -0.00441741943359375, -0.428466796875, -1.3916015625, -2.2890625, -1.310546875, -1.0556640625, 0.46875, 0.1898193359375, 1.099609375, -0.77783203125, 1.47265625, -1.5673828125, -0.59765625, 0.7236328125, -0.9873046875, 1.021484375, -1.818359375, -0.80712890625, 0.75390625, 0.50439453125, 0.794921875, 0.53173828125, -1.9033203125, 1.1025390625, 0.7255859375, -0.0438232421875, 0.25244140625, -0.386474609375, -0.90087890625, -0.8701171875, 1.4892578125, 0.303955078125, -0.479736328125, -0.546875, 1.2880859375, 1.181640625, 1.009765625, 1.2587890625, 0.58251953125, 1.1357421875, -0.0262603759765625, -0.62744140625, -1.4755859375, -0.609375, 0.7255859375, 0.625, -0.408935546875, -0.50390625, -2.33203125, 1.4443359375, -0.9208984375, -1.09765625, 0.576171875, -1.64453125, -1.380859375, 0.453369140625, 0.7392578125, 1.1123046875, -1.4765625, -0.044219970703125, -0.03265380859375, -0.69189453125, 0.80029296875, 0.5302734375, 1.4609375, 0.499267578125, 2.669921875, -0.83349609375, -1.125, -2.890625, 0.6923828125, -0.004192352294921875, 1.2265625, 0.451171875, 0.343505859375, -0.5537109375, 0.79736328125, 1.4892578125, -1.1904296875, -2.28515625, -0.143798828125, 0.8642578125, -0.61669921875, 0.469970703125, -0.2259521484375, -1.162109375, -0.34423828125, 0.0927734375, -0.9033203125, 0.9814453125, 0.039764404296875, 0.430419921875, 1.1806640625, 0.37451171875, -1.0234375, -1.1181640625, -0.37646484375, -2.0859375, 0.83837890625, -0.407958984375, -1.095703125, -0.1434326171875, 0.62451171875, 0.295166015625, 0.95166015625, -0.283203125, 1.5, 1.337890625, -0.0977783203125, -0.55810546875, 1.1572265625, 2.40625, 1.1474609375, 1.369140625, -1.3017578125, -0.58251953125, -0.364990234375, 1.3525390625, -1.3330078125, -0.359619140625, 0.335693359375, 1.5048828125, 2.279296875, -0.497314453125, -0.1314697265625, -0.4443359375, -0.9248046875, 2.642578125, 0.51904296875, 0.0177459716796875, 0.387451171875, 0.393798828125, 1.0810546875, -0.8388671875, -0.1661376953125, 0.43896484375, -1.1240234375, 0.42236328125, -0.34375, 1.0654296875, 0.1702880859375, -1.7138671875, -0.0014295578002929688, 0.57958984375, 0.032989501953125, 0.086669921875, -0.0164947509765625, 1.6220703125, -0.935546875, -0.1376953125, -0.134521484375, 0.52099609375, 0.091796875, 0.71875, 0.4794921875, -0.86376953125, 0.167724609375, 0.2347412109375, 2.908203125, -1.904296875, 1.33203125, 0.1864013671875, 0.419921875, 0.1588134765625, 0.62451171875, 0.9658203125, 0.2093505859375, 0.275146484375, 1.1064453125, 0.265625, 0.66845703125, 0.857421875, -0.39697265625, -1.1416015625, 0.93505859375, -0.875, 0.11553955078125, -0.099365234375, 0.028472900390625, -0.99072265625, -0.039794921875, -0.513671875, 1.1552734375, 1.1953125, -1.662109375, 0.41162109375, 0.7119140625, 0.63818359375, 0.8466796875, -1.705078125, 0.78564453125, 0.85498046875, -0.181396484375, 0.309814453125, -0.5654296875, 0.607421875, 0.2099609375, 0.57275390625, -0.19677734375, 1.796875, -0.7060546875, -2.02734375, -0.96923828125, -0.036468505859375, -0.32275390625, 1.19921875, 0.10284423828125, 0.88427734375, 0.73974609375, 0.062255859375, 0.61767578125, 0.11932373046875, 0.157470703125, -0.181884765625, -0.25732421875, 1.826171875, -2.8046875, -1.1923828125, 1.212890625, 0.184326171875, 1.40234375, 0.354736328125, -0.9658203125, 0.1358642578125, -0.35107421875, 2.1171875, -1.1484375, -1.17578125, 2.06640625, 0.2392578125, -1.62890625, 0.91162109375, -0.251220703125, -0.8828125, -1.7744140625, 0.305419921875, 0.9541015625, 0.7919921875, -1.0791015625, 0.732421875, -0.37109375, -0.818359375, -1.0546875, 1.23828125, -0.1632080078125, 0.9453125, -0.75048828125, 0.04669189453125, -0.214111328125, 0.421142578125, 2.04296875, 0.5498046875, -0.607421875, 2.2265625, -0.8583984375, -1.158203125, 0.77392578125, -0.69775390625, -0.79150390625, -1.271484375, 1.26953125, -0.283447265625, -1.263671875, 0.6181640625, -0.892578125, -0.76025390625, -0.49560546875, 0.361572265625, -0.1058349609375, 0.2489013671875, 1.0048828125, -1.08984375, 1.3046875, 0.64990234375, 0.09039306640625, -0.365966796875, -0.2032470703125, 0.66064453125, 0.80859375, 0.71337890625, 0.3671875, 1.3115234375, 0.87255859375, -1.1357421875, 1.3125, 0.78857421875, 0.9970703125, -0.06646728515625, 0.04608154296875, -0.100341796875, 0.112060546875, -0.98583984375, -0.01117706298828125, 0.98193359375, -1.849609375, 0.95361328125, 0.35009765625, 1.619140625, -0.96044921875, -0.9765625, 1.4658203125, 0.04864501953125, 1.623046875, 0.60302734375, 0.90234375, -0.51806640625, 0.289306640625, -0.5, -0.72802734375, -0.322509765625, -0.7099609375, -1.2138671875, 1.0517578125, 0.34326171875, -0.74951171875, 0.339111328125, 0.80322265625, -0.4072265625, 2.572265625, 0.98974609375, 1.03125, 0.90673828125, -0.8955078125, 1.087890625, 0.36376953125, -1.359375, -0.90576171875, -0.397705078125, 0.297119140625, 0.85595703125, -0.06793212890625, -0.806640625, -1.03515625, 0.11273193359375, -1.0703125, -1.8125, -0.1470947265625, -0.51123046875, -0.1435546875, 0.257080078125, -2.171875, 0.466552734375, 0.1519775390625, -1.001953125, -1.7275390625, 1.126953125, 1.021484375, 0.490234375, 1.0693359375, -0.99609375, -1.0927734375, -0.292724609375, 0.62353515625, -0.66552734375, 1.060546875, 1.697265625, 0.68994140625, -0.19287109375, 0.0015382766723632812, 0.28369140625, 0.70556640625, -2.001953125, -1.3466796875, -1.6337890625, -0.424560546875, -0.311767578125, 0.317626953125, 0.351806640625, 0.329345703125, -1.7109375, 1.8544921875, 0.1907958984375, 1.0322265625, 0.11553955078125, -0.299072265625, -0.50439453125, 2.548828125, -1.8212890625, 1.39453125, 1.7998046875, 0.50830078125, -1.1796875, 0.205322265625, -0.98193359375, -0.157958984375, 0.9404296875, -0.6083984375, 0.24462890625, -0.9462890625, -1.2529296875, 1.3896484375, -1.109375, 0.107421875, 0.440185546875, -0.2266845703125, 0.39794921875, 2.326171875, 0.60009765625, -0.724609375, -2.326171875, 0.6396484375, 0.2474365234375, 0.275390625, 0.48291015625, -0.7626953125, -1.38671875, -0.0665283203125, 0.07342529296875, -0.7666015625, -0.7099609375, 0.05682373046875, -0.79248046875, 0.281005859375, -0.685546875, 0.59619140625, -0.4248046875, -0.796875, 0.0125732421875, -0.07977294921875, -0.72265625, 0.53662109375, -0.3525390625, -1.6328125, 0.85302734375, 0.5751953125, 0.98046875, -0.347412109375, -0.6220703125, -1.01953125, -0.6533203125, -1.0654296875, -0.261962890625, 1.244140625, 2.560546875, 0.78271484375, 0.80517578125, 1.1171875, 1.396484375, 1.1748046875, 0.09954833984375, -1.2578125, -0.03369140625, 0.56787109375, 0.64111328125, 0.525390625, 1.1396484375, -0.017242431640625, -1.123046875, 0.47900390625, 1.0087890625, 0.6845703125, 0.6259765625, -0.245361328125, 0.1405029296875, -0.6787109375, -0.55615234375, 0.63818359375, -0.215576171875, 0.7890625, -0.27294921875, -2.400390625, -0.7919921875, -0.1727294921875, 0.5390625, -1.59765625, 0.86328125, 0.474853515625, 0.06866455078125, -0.31396484375, 1.44140625, 0.11700439453125, 0.96875, 1.1572265625, 1.80859375, -2.23828125, 0.32275390625, 1.9384765625, 0.8037109375, 0.453369140625, 1.744140625, 0.54541015625, -0.75244140625, 1.2216796875, -0.99072265625, 1.392578125, 1.0888671875, -0.498046875, -0.0160980224609375, 1.1240234375, -1.8857421875, -1.3515625, 0.72021484375, -1.421875, 0.525390625, -0.1263427734375, 1.7265625, 1.6923828125, 0.4423828125, 0.81689453125, -0.76025390625, -0.6259765625, 1.0126953125, -0.256103515625, 1.7177734375, 0.548828125, 0.82421875, 0.65087890625, -1.732421875, -1.40234375, -0.4677734375, -0.74365234375, 2.142578125, 0.7119140625, -0.73291015625, -1.8818359375, 1.5517578125, 1.505859375, -0.0004324913024902344, -1.1806640625, -0.5283203125, -0.76220703125, -1.1640625, 0.81640625, 1.2568359375, -1.4658203125, -0.970703125, -0.5185546875, -1.3115234375, -0.459228515625, -0.46337890625, 0.931640625, 0.38916015625, 0.5634765625, -0.61767578125, -1.2783203125, -0.68212890625, -0.59130859375, 1.0126953125, -0.1151123046875, 0.369384765625, -0.291015625, 0.0099945068359375, -1.234375, -0.396240234375, 0.7353515625, 0.433837890625, 2.2734375, 1.3388671875, -0.76806640625, 0.0567626953125, 1.1396484375, -0.37109375, -0.1695556640625, 0.2095947265625, 1.05078125, -0.1500244140625, -0.446044921875, 1.275390625, -0.366943359375, 1.2685546875, -0.677734375, -0.2213134765625, 0.09149169921875, 0.64697265625, 0.477783203125, -0.09332275390625, 2.46875, 0.76318359375, 0.135009765625, -0.95068359375, -1.5009765625, -0.5205078125, 2.248046875, -0.70458984375, 0.11865234375, 0.11346435546875, -0.75830078125, -0.83984375, 1.859375, -0.3623046875, -1.8818359375, -1.7451171875, 1.4267578125, 0.1348876953125, -1.7802734375, -0.286376953125, 0.254150390625, -0.54443359375, -2.146484375, 0.78466796875, -0.1832275390625, 0.1878662109375, 1.2470703125, 0.84326171875, -0.544921875, 0.221923828125, -0.044952392578125, -0.400146484375, 0.59716796875, 0.11212158203125, -0.5361328125, -0.2481689453125, 1.4658203125, -2.3125, 0.45068359375, -0.41748046875, 0.419189453125, 1.2041015625, 0.86865234375, -0.5966796875, -0.60888671875, -0.5068359375, -0.240234375, -0.08929443359375, -0.85400390625, -0.220458984375, -0.053253173828125, 0.201904296875, -0.755859375, -0.160400390625, -0.28125, -0.734375, -0.497802734375, -0.08880615234375, 0.0309906005859375, -0.488037109375, 2.111328125, -0.1793212890625, -0.08575439453125, -2.962890625, 0.81005859375, -0.76611328125, 0.7548828125, 0.4921875, 0.2359619140625, -0.14208984375, 1.4521484375, 1.5546875, 0.1856689453125, -1.6396484375, -0.904296875, -0.4765625, 1.2197265625, -0.52685546875, -0.8271484375, -1.2451171875, 1.23046875, -1.3828125, -0.04510498046875, -0.333740234375, 0.455322265625, 0.60302734375, -1.6982421875, 1.615234375, -0.6083984375, 0.19287109375, -0.478271484375, -1.3642578125, -0.89306640625, 1.0546875, -0.1689453125, -3.5078125, 0.431396484375, -0.326904296875, -0.43505859375, -0.3916015625, 0.56640625, 0.814453125, 0.62353515625, -0.3935546875, -0.491455078125, 0.36181640625, -1.6376953125, -0.1793212890625, -0.6025390625, -1.4248046875, -0.090087890625, 0.49658203125, -0.515625, 1.1005859375, 0.1822509765625, 0.96435546875, 0.5576171875, -0.1309814453125, 0.32421875, 1.7587890625, -0.27880859375, -1.154296875, 0.152099609375, -1.37109375, -1.2119140625, 0.197998046875, -0.143310546875, -1.53125, 0.7578125, 0.081298828125, -1.119140625, -0.303955078125, 0.1783447265625, 0.2059326171875, 0.228759765625, -1.447265625, -1.1044921875, 2.3671875, 1.234375, -0.9794921875, 1.396484375, 0.230224609375, -1.7919921875, 0.3623046875, 0.478759765625, -1.466796875, 1.537109375, 0.267333984375, 0.7333984375, 0.572265625, 0.94482421875, 0.50732421875, -0.705078125, -0.95947265625, -0.56591796875, 1.2685546875, -0.44873046875, 0.630859375, -0.56103515625, -0.486328125, 0.20458984375, -0.9033203125, -1.115234375, -0.56591796875, -0.7392578125, 0.83544921875, -0.64599609375, 1.0283203125, 0.64794921875, 0.88134765625, 0.309814453125, 0.416015625, 1.2880859375, 1.998046875, -1.2119140625, -2.8203125, -0.82421875, 0.061431884765625, -1.982421875, -0.11669921875, -0.10736083984375, 0.228759765625, -0.248779296875, 0.41796875, -1.24609375, 0.00011676549911499023, 1.3779296875, 0.55126953125, -0.75439453125, -0.28955078125, 1.4794921875, -1.548828125, -1.26171875, 0.58154296875, 0.12548828125, 0.42822265625, -0.5185546875, 0.431884765625, 1.7822265625, -0.80712890625, -0.52587890625, -0.76806640625, 0.01910400390625, 1.7763671875, 0.2005615234375, 0.66796875, -1.9140625, -0.64208984375, -0.5859375, 0.15478515625, -1.12109375, -2.466796875, 0.474365234375, 1.1591796875, 0.20703125, -0.318115234375, 0.0274810791015625, -1.7744140625, 0.80126953125, -0.869140625, -0.26513671875, 0.287109375, 0.62109375, -0.82421875, 0.62451171875, 0.77197265625, -0.587890625, -0.91650390625, -1.9521484375, -0.10638427734375, -0.404052734375, 1.498046875, -0.2027587890625, 0.58203125, -0.325439453125, 1.517578125, -1.3203125, 0.87255859375, 0.99365234375, -1.8564453125, 0.59033203125, -1.154296875, -0.0182647705078125, -0.8466796875, -1.58984375, 0.693359375, -0.69921875, 1.99609375, -0.6240234375, -0.4833984375, -0.7509765625, -0.85986328125, 0.7412109375, -1.240234375, -1.2724609375, -0.0657958984375, -0.66357421875, 0.130126953125, -0.958984375, 2.646484375, -1.115234375, -0.28857421875, 0.0738525390625, -0.51708984375, -0.0167083740234375, 1.396484375, 0.292236328125, 0.14404296875, -1.4404296875, 1.0341796875, 0.0240020751953125, 0.279296875, -0.7001953125, 2.2421875, 0.4638671875, 0.6474609375, -1.8291015625, -0.68505859375, 2.001953125, -0.87451171875, -0.08367919921875, -0.5361328125, 0.35986328125, 1.2265625, 1.69921875, 0.689453125, 0.6630859375, -1.5771484375, 1.2734375, -0.493408203125, 1.1962890625, -1.9697265625, 0.20263671875, 0.251953125, -0.2056884765625, -0.828125, -1.103515625, 0.90966796875, 0.50537109375, -0.447998046875, 0.8623046875, -0.387451171875, 0.716796875, -0.437255859375, 0.7900390625, 1.3720703125, -1.052734375, -1.421875, 1.259765625, -0.197998046875, -1.80078125, 1.162109375, -1.626953125, -0.56103515625, 0.8701171875, 1.41015625, -1.5009765625, 0.1231689453125, -0.92919921875, -0.6455078125, -0.394287109375, -0.257080078125, 2.01953125, 0.96484375, 0.99853515625, -1.4462890625, 0.3740234375, -1.3837890625, -2.052734375, -0.259521484375, 1.1396484375, 2.140625, -0.332275390625, -0.625, -0.72998046875, -0.207275390625, -2.087890625, 0.41650390625, -0.1492919921875, 0.0222015380859375, 1.4228515625, -1.5078125, -0.3544921875, 0.8251953125, -0.12103271484375, 1.2880859375, 1.2548828125, 0.52783203125, -0.89990234375, 0.301025390625, -1.017578125, 0.3896484375, -0.1318359375, 1.736328125, -0.33642578125, -0.2274169921875, 1.443359375, 0.9697265625, -1.0302734375, -0.219482421875, -0.697265625, 1.0048828125, 0.82861328125, 0.748046875, -0.042022705078125, -1.138671875, -1.431640625, 0.1241455078125, -0.859375, 0.0158538818359375, -1.025390625, -0.75, -0.5634765625, 1.068359375, -0.6943359375, -1.728515625, 0.64697265625, 1.8056640625, -1.1943359375, -0.27490234375, 0.82080078125, 0.734375, -0.0309600830078125, 0.5390625, 0.98876953125, -1.0107421875, -0.1712646484375, 0.9541015625, 1.8203125, 0.98876953125, -1.5419921875, -1.4794921875, 1.1025390625, 0.646484375, 0.092529296875, 0.1575927734375, -0.302734375, 0.52001953125, -1.2919921875, -0.03936767578125, -0.68994140625, -0.10516357421875, -0.962890625, 0.93701171875, 1.4501953125, -0.464111328125, -0.92919921875, -0.1793212890625, 0.2100830078125, -0.313720703125, -1.3486328125, 0.3544921875, -0.179931640625, -0.400390625, 1.71875, -0.191650390625, -1.9111328125, -1.150390625, -0.6533203125, 0.381103515625, 0.455322265625, -0.96337890625, -0.27587890625, 0.0295562744140625, -1.2568359375, -1.2841796875, 0.1458740234375, 0.12322998046875, -0.82275390625, -0.83984375, -0.07635498046875, 0.09356689453125, 1.3681640625, -0.040252685546875, 0.6435546875, 1.384765625, -1.921875, 1.15234375, -0.91845703125, -0.4365234375, 0.348388671875, -1.83984375, 1.298828125, 2.578125, 0.1348876953125, -1.3720703125, 0.397216796875, -0.7607421875, -1.3505859375, 0.021942138671875, -0.05780029296875, 1.3623046875, -0.2958984375, -1.572265625, -1.1884765625, -0.2421875, -1.15234375, -1.296875, -0.37060546875, -1.2158203125, -0.833984375, -0.453125, 0.70556640625, 1.70703125, -2.08203125, -1.2216796875, -0.37353515625, -0.45556640625, -0.052001953125, -1.4580078125, 0.12548828125, 0.2783203125, 1.7626953125, -0.4775390625, 0.471435546875, -0.87255859375, -0.492919921875, 1.923828125, -1.3115234375, -0.91943359375, 2.072265625, 0.277587890625, -0.537109375, -0.7294921875, -0.38232421875, 0.15673828125, 0.30078125, -0.10546875, -1.4716796875, 0.87255859375, -2.10546875, 0.7099609375, -1.125, 0.3251953125, -0.08404541015625, -0.37353515625, 1.185546875, -0.371337890625, -1.2177734375, 0.4423828125, -0.88037109375, 0.72802734375, -1.82421875, 1.4091796875, -1.5224609375, 0.560546875, 0.2152099609375, 0.2454833984375, 1.3095703125, 1.0146484375, 1.171875, -0.015777587890625, 1.1669921875, 0.2125244140625, -0.94287109375, -0.9677734375, 0.58251953125, -0.479248046875, -1.3017578125, -0.5126953125, -0.41015625, 0.5634765625, -0.4189453125, -0.0226593017578125, 0.2332763671875, -0.912109375, 1.673828125, -0.8974609375, 1.263671875, -0.89306640625, -0.490234375, 1.375, 2.3203125, -0.479248046875, 1.7509765625, -0.063232421875, 0.57763671875, 1.2314453125, 1.1650390625, 1.9619140625, -0.90185546875, 0.83349609375, -0.021942138671875, 1.3447265625, -0.324951171875, 0.024993896484375, 0.420166015625, -0.34326171875, 0.859375, 0.64306640625, 1.1943359375, -0.2406005859375, 0.128662109375, -1.90625, 1.4072265625, -0.72607421875, 0.71923828125, 2.427734375, -0.09210205078125, -1.9453125, -2.12109375, -0.6806640625, -1.0859375, 1.2578125, 0.51806640625, -1.4306640625, -1.685546875, -0.73828125, -0.5498046875, 2.783203125, 0.1229248046875, 0.63134765625, 0.494384765625, 1.0693359375, 0.0204010009765625, 0.47314453125, 0.311767578125, -0.050445556640625, -1.8603515625, 0.483154296875, 0.90283203125, -0.81787109375, -0.400146484375, -1.1572265625, -0.341796875, 0.06573486328125, -0.85302734375, 1.2373046875, -2.39453125, -0.30224609375, 0.424072265625, -0.1558837890625, 0.93017578125, 0.896484375, -0.263427734375, 0.69970703125, 0.4853515625, 0.226318359375, 0.27197265625, -1.638671875, 1.626953125, -1.9482421875, -0.332275390625, -0.98486328125, 0.5908203125, -0.127685546875, 0.074951171875, -1.658203125, -0.6904296875, -0.318603515625, -0.7626953125, 0.515625, 1.015625, 0.08526611328125, 0.041412353515625, 0.353759765625, -0.1688232421875, 0.031768798828125, -1.525390625, 0.9814453125, 0.201904296875, 1.1240234375, 0.15576171875, 1.24609375, 0.60400390625, -0.67578125, -0.138671875, -0.65283203125, 0.09478759765625, 0.298828125, -1.748046875, -0.0906982421875, 0.8759765625, 0.2479248046875, 1.5107421875, 0.9248046875, 0.048583984375, -0.51806640625, 0.2215576171875, -1.21484375, -1.0244140625, 1.4384765625, -0.2646484375, -0.9072265625, 1.36328125, -1.3212890625, -1.3125, 1.3330078125, 0.60400390625, 0.96337890625, -1.40234375, -0.1517333984375, 0.1351318359375, -1.7978515625, -0.0802001953125, 0.09564208984375, -0.12744140625, 0.935546875, 1.0166015625, -1.13671875, 1.2451171875, -0.467041015625, -1.236328125, -0.8544921875, -1.0927734375, 0.685546875, -1.60546875, -0.5234375, 0.82373046875, 1.173828125, -0.30224609375, -0.36474609375, 0.697265625, -0.7099609375, 0.80908203125, -1.349609375, -0.344970703125, 0.60791015625, -0.5166015625, -0.0966796875, -0.67919921875, -1.04296875, -1.845703125, 1.5341796875, 0.44482421875, 0.3056640625, -0.5673828125, 0.52978515625, -0.046539306640625, 1.2177734375, 0.75537109375, -0.339599609375, -0.810546875, 1.3671875, -1.6396484375, 1.609375, 1.244140625, 0.94921875, 1.8388671875, -0.228515625, -0.348388671875, -0.390869140625, -0.390625, 0.482177734375, 0.572265625, 0.5400390625, 0.45654296875, 0.7099609375, 1.0615234375, 1.51171875, 0.486328125, -0.05413818359375, -0.1622314453125, 0.4462890625, -0.303466796875, 0.300537109375, -0.501953125, -0.2335205078125, 1.298828125, -0.60595703125, -0.6552734375, -0.56591796875, -1.5146484375, 0.8271484375, 0.6044921875, 0.5185546875, 0.62451171875, -0.01274871826171875, 0.478515625, -1.0234375, -1.2666015625, 0.271728515625, 0.398193359375, -0.04248046875, 0.48095703125, 0.92138671875, -1.26953125, -0.426513671875, -0.46484375, -0.31640625, 0.335205078125, -0.0511474609375, -1.3349609375, 1.099609375, -0.408203125, 2.228515625, -0.406494140625, -0.6767578125, -0.31640625, -0.1212158203125, -0.014556884765625, -1.0439453125, 1.1171875, -0.5234375, 0.232666015625, -2.12109375, 1.91015625, -0.8173828125, -0.66357421875, -0.80224609375, -0.0165252685546875, 0.65234375, -0.109375, 0.93310546875, -0.419677734375, 0.98046875, -0.4892578125, -1.041015625, 0.369384765625, 0.260498046875, 0.337890625, 0.6533203125, -0.90234375, -0.2430419921875, 0.69677734375, -0.64306640625, 0.982421875, -0.406982421875, -0.5146484375, -1.845703125, -0.48779296875, -0.2191162109375, 0.58203125, 0.7080078125, 0.5166015625, -1.0068359375, 0.67333984375, 0.00753021240234375, -1.4736328125, 0.06439208984375, -0.5576171875, 0.546875, 1.083984375, -0.82470703125, 0.80029296875, -0.8369140625, 2.26171875, -0.1002197265625, 0.7255859375, 0.466796875, -1.15625, 0.82177734375, 1.1025390625, 0.5498046875, -0.525390625, 2.13671875, 0.60546875, -0.32470703125, 0.1129150390625, -0.6123046875, -0.06365966796875, 0.31396484375, -1.3798828125, 1.970703125, -1.4833984375, -2.369140625, 0.87939453125, -0.87255859375, -0.66650390625, 0.1842041015625, -1.994140625, 0.9228515625, -0.7685546875, 1.1474609375, 1.21875, 0.322265625, -0.283203125, 0.55712890625, -1.2880859375, 0.345947265625, 0.371826171875, -0.5791015625, 1.666015625, -0.52587890625, -1.1103515625, 0.2275390625, -0.428955078125, -0.36376953125, 0.6220703125, -0.8173828125, 0.1729736328125, -0.77197265625, -0.054962158203125, 0.818359375, -0.255859375, 1.2509765625, 0.46435546875, -1.7021484375, -1.5576171875, 0.60986328125, -1.0927734375, 0.0926513671875, 0.658203125, 0.08624267578125, 0.888671875, -1.857421875, 1.2841796875, 0.7607421875, -0.9892578125, 0.6865234375, 1.2177734375, -1.05859375, -0.8447265625, -0.64501953125, -0.9013671875, -1.3203125, 0.60546875, -2.876953125, 0.96826171875, 0.3037109375, -0.2186279296875, 0.221923828125, -1.43359375, 0.8359375, 1.4404296875, -0.06378173828125, -0.95751953125, -1.4453125, 1.0439453125, -1.6982421875, -0.200439453125, -0.0791015625, 0.302978515625, -0.007595062255859375, 0.327392578125, -0.37060546875, -1.435546875, -0.51123046875, -1.2138671875, -0.529296875, -1.8525390625, -1.1923828125, -2.0390625, 1.8740234375, 0.609375, -1.515625, -0.165283203125, 1.4697265625, 0.53515625, 0.0263671875, -0.0085906982421875, 2.169921875, 0.1307373046875, -0.45947265625, -0.279296875, -0.250732421875, 0.34130859375, -0.30712890625, 2.39453125, -1.78515625, -0.18896484375, -0.75048828125, -0.52001953125, -0.042755126953125, -0.849609375, -0.7578125, 0.69677734375, 1.087890625, 0.33447265625, 0.92919921875, -0.82373046875, -0.61767578125, -0.572265625, 0.01220703125, -0.91943359375, -1.9013671875, -0.298583984375, 1.3525390625, -0.58837890625, 1.4990234375, -0.2232666015625, -0.235107421875, 0.34228515625, -0.181884765625, 0.319091796875, 1.81640625, 0.69482421875, -0.11798095703125, 1.1630859375, 0.434814453125, -0.763671875, 0.3671875, -1.296875, -0.09576416015625, -1.7578125, 1.7607421875, -0.1097412109375, -0.99560546875, 2.3125, 1.0625, -0.5576171875, -0.25390625, -0.209716796875, -1.9189453125, 1.060546875, -0.21142578125, 2.541015625, 0.341064453125, -1.916015625, -0.0635986328125, -1.34375, -0.92333984375, 0.197998046875, 0.2408447265625, -0.62744140625, -1.271484375, -0.2066650390625, 0.51513671875, -0.61865234375, 0.382080078125, 0.4697265625, -0.57275390625, 0.5478515625, -0.1685791015625, -1.1513671875, -0.04986572265625, -1.0078125, -1.791015625, 0.61083984375, 0.505859375, 0.93017578125, -0.32666015625, 0.369873046875, -0.0390625, 0.2081298828125, -2.552734375, -0.1439208984375, -1.119140625, -0.366455078125, 0.93798828125, 1.2255859375, -0.442138671875, -0.78466796875, -1.189453125, -1.3125, 0.77685546875, 0.297119140625, 1.6162109375, 1.240234375, -3.400390625, 0.29150390625, -0.08172607421875, 1.85546875, -1.404296875, -0.556640625, -0.56494140625, 2.173828125, 1.4931640625, -0.63330078125, -0.62548828125, -0.283935546875, -0.544921875, -2.623046875, 0.401123046875, 0.357421875, -0.0135498046875, 0.063720703125, -0.032135009765625, -0.329345703125, -1.0185546875, -0.33349609375, 0.4296875, 1.3515625, 0.41552734375, 1.7275390625, -0.34326171875, 0.07366943359375, 0.57080078125, -0.51318359375, 0.2479248046875, 0.471923828125, -1.5634765625, 0.66796875, 0.07818603515625, 1.515625, 0.131103515625, -0.787109375, 0.78955078125, -0.81689453125, -1.4921875, 1.845703125, 0.58203125, -0.29443359375, -0.7353515625, 1.94921875, -0.12060546875, 1.2236328125, 0.486083984375, -0.472412109375, -1.4775390625, -1.1240234375, 0.5439453125, -0.5126953125, -0.68994140625, -0.58447265625, 1.3828125, 1.693359375, 1.8876953125, 0.397705078125, -0.6611328125, 0.263427734375, -1.171875, 1.474609375, -0.038818359375, 0.298583984375, -0.9697265625, -2.0, 0.7265625, -1.5478515625, -0.363037109375, 0.224365234375, -0.451904296875, 1.0615234375, 0.72021484375, 0.45263671875, 0.2366943359375, 1.2470703125, -0.40087890625, 0.388671875, 1.2841796875, -1.4169921875, -1.4677734375, 1.6767578125, -1.115234375, -0.86962890625, -1.4501953125, -0.8388671875, -1.45703125, -0.8154296875, 0.9326171875, -0.32666015625, -1.6064453125, -0.1678466796875, -0.1668701171875, 0.415771484375, 0.88037109375, -0.58740234375, -0.81640625, 0.270263671875, -0.76025390625, -0.3408203125, 2.72265625, -0.9384765625, -0.8046875, 0.515625, 0.1949462890625, -0.51416015625, 0.467529296875, 0.1622314453125, 0.5009765625, -1.138671875, -0.0657958984375, 0.73583984375, -0.31884765625, 0.380126953125, 1.4208984375, 1.0625, 1.390625, -0.439208984375, 2.08984375, 0.490478515625, -0.43359375, -0.455322265625, -0.49072265625, -0.1788330078125, 0.27001953125, -0.336181640625, 0.64013671875, -1.123046875, -0.986328125, -0.0845947265625, -0.5458984375, 1.189453125, 0.70458984375, 0.501953125, 0.36328125, 0.1944580078125, 0.0252227783203125, -0.351806640625, 0.54443359375, -1.67578125, 1.9423828125, 0.76708984375, -0.496826171875, 1.8359375, 1.3544921875, -0.04058837890625, 0.794921875, 0.1932373046875, -0.87744140625, 0.25927734375, -0.476318359375, 0.195556640625, -1.3701171875, 0.8935546875, 0.89404296875, 1.3544921875, -0.79443359375, -0.16259765625, -1.9658203125, -0.89794921875, -2.744140625, -0.5615234375, -0.38525390625, -1.4375, 0.329345703125, -0.93798828125, -0.1671142578125, -1.26171875, -1.9326171875, 0.1951904296875, 0.41455078125, -0.09619140625, -0.68603515625, 0.6669921875, 0.303466796875, -0.91845703125, 1.6904296875, -0.501953125, -0.431640625, 1.4228515625, 1.4921875, -0.2529296875, -0.986328125, 0.1519775390625, -0.50341796875, -1.4677734375, -0.417236328125, -0.314453125, 1.0791015625, 1.4951171875, -0.055877685546875, -0.348388671875, 0.7265625, 2.08984375, -1.322265625, -2.109375, -2.51953125, -0.859375, 1.16796875, -1.4951171875, 0.0997314453125, -0.0361328125, -0.76123046875, -0.54541015625, 0.350830078125, 1.201171875, -0.386962890625, -0.40576171875, -1.880859375, -1.0234375, 0.487060546875, -0.3056640625, -0.172119140625, -0.296630859375, 0.68212890625, -3.62890625, 1.0185546875, 1.3291015625, -0.289306640625, -0.30126953125, -0.93505859375, 1.115234375, 1.0087890625, 0.99658203125, -0.20654296875, -0.383544921875, 0.69677734375, 0.7978515625, 0.962890625, -1.2177734375, 0.7744140625, 1.7744140625, 0.6591796875, -0.266845703125, 0.759765625, -1.8671875, -0.222900390625, 0.1787109375, 0.68310546875, -0.06732177734375, 0.416015625, -0.046234130859375, -0.470703125, 0.277587890625, -0.27294921875, -0.89990234375, -1.3671875, 0.80322265625, 0.53466796875, 1.26171875, 0.1307373046875, 1.3955078125, -0.162841796875, -0.75146484375, 1.568359375, 1.1396484375, 1.2119140625, 0.485107421875, 1.6455078125, -0.1124267578125, -0.274169921875, 0.455810546875, 1.0244140625, 0.10748291015625, -0.7353515625, -0.15625, -0.89990234375, -1.1787109375, 0.0030612945556640625, -0.3544921875, -0.2861328125, 1.0009765625, 1.1806640625, -0.189208984375, 0.552734375, 1.0205078125, -0.321533203125, -2.083984375, -1.384765625, -0.14501953125, -1.0546875, 0.0843505859375, 1.5, 1.078125, 1.1552734375, -0.223876953125, 1.1552734375, -0.87109375, 1.2451171875, 1.0224609375, -1.904296875, 0.32373046875, -0.619140625, 2.2734375, -0.137451171875, 0.223876953125, 1.443359375, 0.04632568359375, -0.061004638671875, -0.244140625, -0.57177734375, 0.701171875, -2.064453125, -0.9501953125, 0.8466796875, 0.4267578125, 0.362548828125, 0.5009765625, 0.01412200927734375, -1.828125, 0.73095703125, 0.040557861328125, 1.603515625, -0.21484375, -0.1868896484375, 0.281982421875, -1.51953125, -0.2000732421875, 1.8291015625, 1.626953125, 0.99365234375, -1.5, 0.2236328125, 0.250244140625, -1.763671875, -1.8154296875, 0.9326171875, 1.3798828125, 1.4296875, -0.32861328125, -0.74853515625, 1.001953125, -0.1534423828125, 0.43408203125, 0.5693359375, -1.462890625, 1.095703125, 1.6845703125, 0.78271484375, -0.045440673828125, 0.4560546875, 0.1671142578125, 0.2215576171875, -1.2109375, -1.384765625, 0.7158203125, 0.321044921875, -0.63525390625, 1.869140625, 0.46630859375, 0.9111328125, -2.2890625, -0.08123779296875, 0.78759765625, -0.10052490234375, 1.083984375, 0.88134765625, -0.92431640625, 0.235595703125, -1.287109375, -1.673828125, 0.53564453125, -3.015625, -0.56787109375, 0.99169921875, -0.8076171875, -1.1669921875, -0.0689697265625, -0.14013671875, 0.044586181640625, -0.92626953125, -0.2685546875, 0.74658203125, 1.0146484375, 0.31884765625, 1.5263671875, -0.77001953125, -1.5126953125, 0.438232421875, -0.73876953125, -0.4951171875, -0.167236328125, 1.5751953125, -0.48583984375, -1.689453125, -0.82763671875, 0.73583984375, 0.1029052734375, 0.41650390625, -0.1900634765625, -0.413818359375, 0.87109375, -0.07098388671875, 1.2734375, 0.7236328125, -0.125732421875, -0.379638671875, 0.61767578125, 0.122314453125, -0.41845703125, 1.345703125, 1.205078125, 0.818359375, 2.13671875, 0.8984375, -0.01129150390625, -2.126953125, 0.5068359375, 0.1795654296875, -1.4345703125, -0.66357421875, -1.7744140625, -1.7578125, -0.52099609375, -1.7333984375, 0.72119140625, -0.2244873046875, -1.416015625, 0.77880859375, 0.122802734375, -1.6513671875, 1.2890625, 1.029296875, 0.3779296875, -0.455322265625, -0.79931640625, -0.185791015625, 2.48046875, -2.056640625, -0.62939453125, -1.09765625, -0.98779296875, 0.11181640625, -0.2235107421875, -0.12939453125, 0.038238525390625, -0.061279296875, 0.8720703125, 0.73828125, 0.394287109375, -0.74365234375, 0.2008056640625, -1.521484375, 0.1795654296875, -1.076171875, -0.8798828125, -0.44482421875, -0.385009765625, 1.044921875, -1.1904296875, 0.357666015625, -0.3408203125, 1.15625, -1.4990234375, 1.1796875, -0.25341796875, 1.37890625, 0.86669921875, 0.059600830078125, -0.51904296875, -0.9814453125, 1.6943359375, 0.9814453125, -0.6484375, 0.7236328125, 0.86669921875, 0.1505126953125, -0.11236572265625, -0.00608062744140625, -0.337890625, -0.1295166015625, 0.3203125, -0.99462890625, 0.8818359375, 0.34912109375, -0.491455078125, -0.548828125, 1.720703125, 0.007171630859375, -1.5615234375, 0.427734375, 0.93505859375, 0.08721923828125, 1.796875, -0.3857421875, 0.70654296875, 0.97509765625, -1.4208984375, 0.6220703125, -0.71240234375, 0.427490234375, -0.228515625, -2.3515625, 0.477783203125, -1.205078125, 0.445068359375, 0.1910400390625, -1.6962890625, -1.275390625, 0.9052734375, 1.6865234375, 0.08856201171875, -0.67822265625, -0.2164306640625, 2.21875, -0.41845703125, -1.9130859375, -0.9013671875, 0.173095703125, 1.3046875, -0.8037109375, -0.478515625, 1.5419921875, 0.8134765625, 0.419677734375, -0.341064453125, 0.95068359375, 0.67236328125, 0.444580078125, 0.399658203125, -0.301025390625, -1.416015625, 0.97607421875, -0.69287109375, -2.642578125, -1.97265625, 1.3701171875, -2.181640625, -0.62939453125, 0.017486572265625, -1.0283203125, -0.435546875, 0.525390625, -1.787109375, -0.56640625, 0.5869140625, 0.60546875, -1.46875, 1.6611328125, 0.424560546875, 1.677734375, -1.6123046875, 0.74365234375, -0.6376953125, 0.6728515625, -0.471435546875, -1.4072265625, 0.548828125, 0.70947265625, -0.1641845703125, -1.6611328125, 0.469482421875, -0.235107421875, 2.291015625, -1.455078125, -0.315673828125, -0.609375, 1.2353515625, -1.1201171875, -1.7587890625, -0.412109375, 0.2030029296875, -0.13720703125, -2.408203125, -1.3173828125, -0.288818359375, 0.9287109375, -0.60791015625, 0.046295166015625, 0.78515625, -0.40966796875, 0.3427734375, -0.08251953125, -0.247314453125, -1.4931640625, 0.1531982421875, 0.83984375, 0.1468505859375, -0.368896484375, 0.6904296875, 0.93408203125, 0.219970703125, -1.166015625, -0.60498046875, -0.83447265625, -1.4345703125, 0.41845703125, -0.2406005859375, 0.427734375, 0.313720703125, 0.94921875, 0.126708984375, -1.8583984375, -0.23583984375, 2.453125, -1.36328125, 1.4443359375, -0.132080078125, 0.1444091796875, -0.86376953125, -0.286376953125, 0.70068359375, 1.138671875, -0.8466796875, 0.6787109375, -1.3876953125, 2.96484375, 1.1083984375, 0.71337890625, -1.46484375, -0.94482421875, -0.9169921875, 2.6015625, -0.2939453125, -0.72119140625, -0.424072265625, -2.90234375, -1.1904296875, -0.0799560546875, 0.1771240234375, 1.1533203125, 0.27978515625, -0.43212890625, 0.07598876953125, -1.3583984375, 0.482421875, 1.275390625, 0.007076263427734375, 0.1077880859375, 1.486328125, -0.167724609375, 0.517578125, -0.79638671875, 0.07818603515625, -1.15234375, -0.4619140625, -0.55078125, 1.0166015625, 1.7978515625, 1.009765625, -0.44384765625, -0.81640625, 2.091796875, 1.7177734375, 1.0361328125, -0.439453125, -0.337158203125, -1.2265625, -1.1474609375, 0.091064453125, 1.53515625, -0.223388671875, 0.42822265625, -1.6484375, 2.013671875, 1.5703125, -1.017578125, -0.145751953125, -1.0810546875, 0.30078125, -1.0107421875, 1.166015625, 0.040557861328125, -0.5771484375, -0.73486328125, 0.391845703125, -0.21923828125, 0.6806640625, -2.4140625, 0.3310546875, 0.025146484375, -0.5634765625, -1.001953125, -0.1678466796875, 0.599609375, 0.1331787109375, 1.08203125, 0.5068359375, -0.65087890625, 1.333984375, 0.98046875, -0.09649658203125, -1.60546875, -0.402099609375, 2.865234375, -1.0693359375, 0.96875, -0.384765625, -0.1693115234375, 0.82080078125, 0.84814453125, 1.205078125, 0.51953125, -1.77734375, 0.301513671875, -0.445068359375, 1.0947265625, -0.50732421875, 1.92578125, -0.167236328125, -0.69384765625, -1.333984375, -0.0908203125, 0.86083984375, 0.68212890625, -1.4443359375, 0.313232421875, 0.08306884765625, -0.1683349609375, 0.66064453125, 0.0755615234375, -0.06597900390625, -0.399658203125, 0.256103515625, 1.9111328125, 1.7861328125, 0.1417236328125, 0.689453125, -0.48779296875, -0.489013671875, 0.79833984375, 1.5654296875, 0.53515625, -1.212890625, -0.050048828125, 1.28125, -0.4697265625, 1.294921875, 1.2880859375, -0.286376953125, -0.85205078125, 0.09185791015625, 0.9423828125, 0.410888671875, -0.41650390625, -0.9765625, 0.38525390625, 0.58154296875, 1.642578125, 0.33642578125, -0.2197265625, 0.396484375, 1.400390625, 0.81396484375, -0.359375, -0.6669921875, 0.0980224609375, -0.11346435546875, 1.0625, 0.8056640625, 0.54736328125, 1.544921875, 0.53076171875, -0.323486328125, 0.92236328125, 0.30029296875, -0.4091796875, -1.1201171875, -0.78662109375, -0.640625, 1.083984375, 0.2408447265625, -0.693359375, 0.1038818359375, 1.2216796875, 1.4287109375, 0.1439208984375, -0.4873046875, -1.6044921875, 0.062225341796875, -0.36474609375, 0.145263671875, -0.17822265625, -0.85888671875, -0.2225341796875, -0.75537109375, -0.0517578125, 0.47705078125, -0.88671875, 2.197265625, -0.375, 0.81298828125, 0.70849609375, 0.0413818359375, -1.078125, -0.71728515625, -1.033203125, 0.01153564453125, 2.013671875, 0.79150390625, 0.2188720703125, 0.465087890625, -0.5556640625, 1.65625, -0.353515625, 0.03936767578125, -0.282958984375, 2.021484375, -1.8857421875, 0.197265625, 0.08868408203125, -0.7216796875, -0.8095703125, 0.3681640625, 1.9853515625, -1.59765625, -0.353515625, -0.1737060546875, -0.474853515625, 1.0830078125, 0.011810302734375, 0.017669677734375, 0.60791015625, -1.041015625, -1.36328125, -0.028717041015625, -1.431640625, -0.5400390625, -0.3681640625, 1.3115234375, -0.783203125, 1.1767578125, -0.8271484375, -0.677734375, -0.21337890625, 0.0221405029296875, 1.9951171875, -0.1214599609375, -0.291748046875, -0.496337890625, -0.06976318359375, -0.953125, -0.07598876953125, 0.73974609375, -0.256591796875, -0.04534912109375, 0.323486328125, 0.69091796875, 1.1884765625, -1.24609375, 1.54296875, 1.7705078125, 0.4462890625, -0.7880859375, -0.473876953125, 0.49755859375, -1.08203125, -1.5009765625, -0.184326171875, -0.405517578125, 0.2073974609375, -0.390625, -0.08038330078125, -0.69970703125, 1.611328125, -0.83837890625, -0.69091796875, 0.60986328125, 0.1402587890625, 0.159912109375, -0.82568359375, 0.5068359375, 2.421875, -0.17626953125, -0.152099609375, 0.405517578125, -1.6279296875, -0.68359375, -0.41259765625, 1.53125, -0.41552734375, 1.4541015625, -1.158203125, -0.07598876953125, -0.338134765625, 0.8515625, -0.626953125, 0.2393798828125, 0.60107421875, -0.243408203125, 0.1756591796875, 1.4296875, 0.211669921875, 1.44140625, -0.32763671875, 0.2308349609375, -0.309814453125, 0.361328125, 0.76513671875, 1.4619140625, 0.5791015625, 2.1640625, -1.517578125, 0.283447265625, -1.9638671875, 1.5576171875, 0.1429443359375, -0.1177978515625, 0.962890625, -0.65625, -1.02734375, -2.1328125, -1.0625, -0.07037353515625, -2.013671875, 1.048828125, 1.0625, 0.36865234375, 1.5791015625, 1.3466796875, -0.0924072265625, 0.623046875, -2.388671875, -0.069580078125, 0.104248046875, -0.97607421875, -0.84912109375, 0.11468505859375, -1.0458984375, 1.2099609375, 1.5068359375, -0.58349609375, -0.67724609375, -0.26513671875, 0.4140625, -0.010040283203125, 1.052734375, -0.143310546875, 0.51025390625, 0.4521484375, 0.1304931640625, 2.03125, 0.1923828125, 0.355712890625, -1.365234375, 0.2056884765625, 0.7666015625, 0.7666015625, 0.374755859375, 2.794921875, 0.9228515625, -0.220947265625, -0.405517578125, -0.0285186767578125, 2.646484375, 0.06500244140625, -1.6826171875, -0.57421875, -0.10784912109375, 1.4248046875, -0.341796875, 1.7626953125, -0.888671875, 0.228271484375, -0.810546875, 2.27734375, -0.1036376953125, -0.822265625, 0.47412109375, 1.134765625, 0.430908203125, 1.6171875, 0.1927490234375, -0.9326171875, -0.853515625, 1.94921875, -0.8232421875, 0.72216796875, 0.89453125, 1.1171875, 0.66259765625, 0.87109375, -1.0322265625, -0.3955078125, -0.018585205078125, -0.07977294921875, -0.58984375, -0.293212890625, 1.146484375, -0.289306640625, -0.55859375, -1.8427734375, 0.1279296875, 1.4248046875, -2.08203125, -0.467041015625, -0.297119140625, 1.5048828125, 0.787109375, -2.126953125, 0.68603515625, -0.45751953125, -0.8876953125, 0.2352294921875, 0.1864013671875, -1.11328125, -0.07049560546875, -0.94482421875, 0.103515625, 1.546875, -2.013671875, 0.053253173828125, 0.2381591796875, -0.32470703125, 0.50439453125, -0.360595703125, 0.78564453125, 1.791015625, 0.343017578125, 0.07275390625, 0.5888671875, -0.5390625, -0.06396484375, -0.5302734375, -0.52587890625, -0.61376953125, -1.4521484375, 0.5791015625, 0.40380859375, 0.90869140625, 0.283935546875, 0.8466796875, 0.10693359375, -0.7236328125, -1.287109375, -1.380859375, -0.1585693359375, -1.8232421875, -1.111328125, -1.478515625, 0.53955078125, -1.384765625, -0.7568359375, 0.435546875, 1.0703125, 1.0703125, -0.00452423095703125, -0.513671875, 0.54052734375, 1.6416015625, 0.69384765625, 1.615234375, 0.153076171875, 0.0987548828125, -0.470947265625, 0.83984375, -0.2802734375, 0.07318115234375, -1.171875, 0.040771484375, -1.0703125, 1.2333984375, 1.7021484375, 1.03515625, -0.1300048828125, 0.5751953125, 2.345703125, -0.8173828125, -0.2083740234375, -0.68701171875, -1.8935546875, 0.646484375, -0.10577392578125, 1.8427734375, -1.2255859375, 0.4921875, 0.279541015625, 0.09027099609375, -1.4775390625, -0.6650390625, -1.67578125, -0.185791015625, 2.185546875, -0.50244140625, -1.3798828125, -0.336181640625, 0.305908203125, 1.390625, -0.0909423828125, 0.13623046875, 1.6572265625, 8.052587509155273e-05, 0.3076171875, 0.548828125, -1.423828125, -0.50732421875, -0.059722900390625, 0.46240234375, -2.03515625, -0.87255859375, -1.044921875, -1.0244140625, 0.0550537109375, 1.51953125, -0.5361328125, 0.3876953125, -0.78515625, -1.177734375, 0.6201171875, -1.1435546875, -0.1817626953125, 0.11724853515625, -1.4150390625, -1.9716796875, 0.60888671875, -0.399658203125, -0.26123046875, 0.027374267578125, -0.1123046875, 0.409423828125, 0.09808349609375, -0.319580078125, 0.8662109375, 0.47509765625, 0.489501953125, 1.0107421875, 1.37109375, -0.79541015625, -1.1279296875, -0.2587890625, 0.8515625, -0.85791015625, 0.2213134765625, 0.04962158203125, 0.1390380859375, -1.5712890625, 0.238037109375, -1.7646484375, 0.382568359375, 0.77978515625, -1.1259765625, 0.2271728515625, 0.87548828125, -0.311279296875, 0.7724609375, -2.15234375, 0.8330078125, 0.1795654296875, -1.3369140625, -1.279296875, -0.50634765625, -1.0166015625, -0.212158203125, -0.83203125, -0.67236328125, 0.1470947265625, -0.0073089599609375, 0.07855224609375, 0.9375, -1.0947265625, 2.169921875, -0.06903076171875, -0.84423828125, -0.2178955078125, 0.171142578125, -1.1474609375, 0.2095947265625, -0.81396484375, 0.63916015625, -1.0966796875, 0.76416015625, 0.92724609375, -1.404296875, -0.2255859375, -1.91796875, -1.705078125, 1.693359375, -1.2021484375, 0.306640625, 1.4775390625, 0.344482421875, 0.0965576171875, -0.6689453125, -1.1923828125, -0.2244873046875, -0.541015625, 0.7373046875, -1.87890625, 1.193359375, -1.6044921875, -0.5830078125, -0.2666015625, 1.58203125, 0.29248046875, 0.91259765625, 0.99853515625, -0.0018472671508789062, -0.3701171875, 0.943359375, 0.2607421875, 1.5966796875, 0.337646484375, -1.9189453125, -1.9052734375, -0.091796875, 0.28955078125, 0.09930419921875, -0.63232421875, -0.64990234375, 0.85546875, 0.8408203125, 0.62060546875, 0.828125, -2.546875, 2.55078125, 0.1456298828125, -0.99462890625, 1.0654296875, -1.484375, 0.128173828125, 0.11456298828125, 0.66455078125, 2.177734375, -1.7509765625, -0.432861328125, -0.92578125, -0.5625, -1.0380859375, 1.029296875, 0.3154296875, -0.471435546875, 0.6201171875, -1.1875, 0.079345703125, -0.1917724609375, 0.37744140625, -1.5126953125, -0.9853515625, 1.3095703125, 0.02801513671875, 1.3857421875, -0.8603515625, -0.035308837890625, -0.2359619140625, 0.179931640625, -0.55615234375, 0.193115234375, -0.217529296875, 1.1318359375, 0.22998046875, 1.1904296875, -0.8583984375, 2.54296875, 0.50341796875, 0.060333251953125, -0.64111328125, -1.1396484375, 0.44677734375, -0.1534423828125, -1.6904296875, 0.74267578125, 2.291015625, -1.90625, -0.54931640625, -1.8359375, -0.148193359375, -0.8076171875, 0.806640625, 0.94677734375, -0.6806640625, -0.66552734375, 1.2958984375, -0.95166015625, 0.320556640625, 0.79931640625, 0.80810546875, 0.0030536651611328125, 0.88671875, 0.2298583984375, -0.63720703125, 0.0526123046875, 1.8515625, -0.319580078125, 0.51318359375, -0.5712890625, -1.0595703125, 1.0634765625, 0.15283203125, 0.34521484375, -1.009765625, -0.252685546875, -0.399658203125, -0.6083984375, -1.39453125, 0.301025390625, -0.293212890625, -0.61962890625, -0.84033203125, -0.5048828125, -0.46875, 0.050537109375, 0.33984375, -0.81494140625, -2.091796875, 1.3974609375, 0.93798828125, 0.28466796875, -1.0576171875, 0.80712890625, 0.75341796875, -0.17236328125, 0.366943359375, 0.050689697265625, 1.5908203125, -0.2469482421875, -0.8388671875, 0.0251007080078125, -0.45556640625, -0.845703125, -1.244140625, 1.2119140625, -1.3916015625, -1.66015625, 0.1529541015625, 1.740234375, -1.705078125, 0.03314208984375, 0.50927734375, 1.41015625, 1.1943359375, 1.310546875, -1.11328125, -0.7978515625, -1.2236328125, -0.53564453125, 0.15234375, -0.1405029296875, -1.5966796875, 1.0166015625, -0.609375, 0.923828125, -0.3955078125, 2.765625, -0.64453125, 0.62451171875, 0.51123046875, 0.397705078125, -0.359619140625, 0.6455078125, 0.2451171875, 1.5068359375, -1.51171875, 1.11328125, 0.73779296875, -2.533203125, 0.384521484375, -0.46533203125, -0.900390625, -0.95166015625, 0.83056640625, -1.431640625, -0.5556640625, 0.265869140625, 0.4150390625, -0.65283203125, 0.451904296875, 2.09765625, -0.88525390625, -1.5546875, 2.02734375, 0.51806640625, 0.52978515625, 1.29296875, 1.1357421875, 0.268310546875, -0.191162109375, 0.10003662109375, 1.234375, 0.2486572265625, 0.64013671875, 0.53369140625, -0.2115478515625, 0.78515625, -0.414306640625, -0.49072265625, 0.75830078125, -0.0008568763732910156, 1.0888671875, 0.469482421875, -1.3408203125, 1.8310546875, -0.537109375, 1.3388671875, 0.609375, 0.290283203125, 0.814453125, -1.0791015625, 0.712890625, -0.54931640625, -0.9677734375, -1.7060546875, 1.6328125, 1.9462890625, 0.7548828125, 0.52587890625, -1.34765625, -0.6953125, 0.1927490234375, 0.2880859375, -1.125, 1.2783203125, 0.81494140625, -0.7734375, 0.94580078125, 1.0615234375, 0.80322265625, 0.7353515625, 0.5712890625, 1.021484375, -0.0176849365234375, 0.86083984375, -2.453125, -0.69873046875, 0.0762939453125, -0.62939453125, 1.361328125, -0.662109375, -0.360107421875, -0.2431640625, 0.1314697265625, 0.50439453125, -0.125, -0.71044921875, 0.8671875, -0.369384765625, 0.4873046875, 2.048828125, -0.190673828125, 0.32958984375, 0.7763671875, 0.65966796875, -0.0291900634765625, -0.83203125, 1.56640625, -2.314453125, -0.083251953125, -1.21484375, -1.126953125, 1.453125, -0.990234375, -1.7607421875, -0.2261962890625, 0.1854248046875, 1.212890625, -1.5654296875, 0.7216796875, 0.2431640625, -0.62841796875, -0.020843505859375, 1.373046875, 0.2119140625, -0.40234375, 0.80322265625, 0.47314453125, -0.039031982421875, 2.95703125, 0.1702880859375, -1.05078125, -0.01111602783203125, -0.7001953125, -0.238525390625, 0.316650390625, -1.1650390625, -0.487060546875, 1.1533203125, 0.1376953125, 1.7763671875, -0.65087890625, 1.4482421875, -0.025360107421875, -0.302001953125, -1.37109375, -0.7490234375, 1.3779296875, 0.45654296875, -0.93896484375, 0.062744140625, 0.04510498046875, -0.56982421875, 0.324951171875, 0.430419921875, 0.1326904296875, 0.1673583984375, 1.345703125, -0.51806640625, -0.3115234375, -2.185546875, -2.0078125, 1.326171875, 0.8837890625, 1.095703125, 0.45849609375, -0.63232421875, 0.611328125, 1.396484375, 1.0517578125, -0.482421875, 1.0009765625, 0.2010498046875, -0.75048828125, 1.693359375, -0.29833984375, 1.673828125, -0.7265625, 0.2978515625, 0.045166015625, -0.76611328125, 0.9345703125, 0.424072265625, -1.4599609375, 0.0217437744140625, -0.3603515625, 0.7568359375, 0.447265625, -2.701171875, 0.30615234375, 0.235595703125, 0.24951171875, 1.0966796875, 0.1429443359375, -0.5986328125, -1.0986328125, -1.1650390625, 0.356689453125, 2.15234375, -0.787109375, 0.256591796875, -1.07421875, 1.0224609375, -2.10546875, 0.32470703125, 0.63037109375, -0.38623046875, 0.86083984375, -1.533203125, 1.037109375, 1.0712890625, -1.2724609375, 0.344970703125, 1.26953125, -0.10101318359375, -0.306396484375, -0.08843994140625, -0.360595703125, 0.55908203125, 1.3388671875, 0.70947265625, 0.8779296875, -0.51708984375, -0.07232666015625, -1.228515625, -0.261474609375, -0.25390625, 0.046051025390625, 0.140380859375, 1.7568359375, 0.003814697265625, -0.14501953125, 1.0888671875, -0.923828125, -1.6875, 0.93408203125, 1.705078125, 0.0931396484375, 0.4599609375, -1.8837890625, 0.1912841796875, 1.142578125, -0.204345703125, 0.7529296875, -0.84423828125, -0.301025390625, 0.890625, -0.260986328125, 0.1534423828125, -0.75439453125, -0.0296173095703125, 0.35986328125, 0.8203125, 0.1207275390625, 0.1903076171875, -1.064453125, -0.923828125, 0.1898193359375, -1.232421875, -1.8779296875, -1.623046875, 0.56982421875, -0.98486328125, 1.177734375, 1.728515625, 1.017578125, -0.81494140625, 2.06640625, 1.9521484375, -0.52880859375, -0.1614990234375, -0.49169921875, -0.5361328125, 0.51513671875, -0.489013671875, 1.9267578125, -0.908203125, -0.02471923828125, -1.431640625, 1.013671875, -0.41650390625, 1.6728515625, 1.525390625, 0.042816162109375, 0.91259765625, 0.045989990234375, -1.5927734375, -0.00946807861328125, 1.4833984375, -0.7314453125, 0.5078125, -2.654296875, -0.76953125, 0.6015625, 0.375244140625, -0.200927734375, 1.7412109375, 2.240234375, -0.2271728515625, -0.316650390625, 0.94677734375, -1.5087890625, -0.2291259765625, -2.28515625, 0.380859375, 0.042236328125, -0.00838470458984375, 1.990234375, 0.312744140625, -0.2333984375, -0.475341796875, -0.5576171875, 0.744140625, -0.50732421875, 0.61279296875, 1.1875, 0.44384765625, 1.12109375, 2.072265625, -1.5361328125, 2.15234375, -1.0947265625, -1.44140625, 1.0693359375, -0.341064453125, 1.5927734375, 1.0087890625, 0.8779296875, -1.0009765625, 0.76171875, 0.51416015625, -0.6318359375, 1.8798828125, 0.0789794921875, -0.28857421875, -0.9775390625, -1.3466796875, 1.7138671875, -0.1317138671875, 0.552734375, -1.462890625, 0.5107421875, -0.1851806640625, 0.8271484375, 1.21875, 1.1953125, 1.0859375, 0.2281494140625, 1.1083984375, 0.06744384765625, 0.206787109375, 0.490966796875, -0.80810546875, 1.07421875, 0.381103515625, -0.62109375, -0.74609375, -0.14453125, 1.2451171875, -0.970703125, 0.26708984375, 1.3720703125, -0.12432861328125, -0.306884765625, 0.5966796875, -1.30078125, 2.021484375, -0.136474609375, -0.94384765625, -0.2227783203125, -0.58935546875, -1.943359375, -1.0126953125, -1.0537109375, -0.052886962890625, -0.499267578125, -1.482421875, -0.5888671875, 0.4287109375, 0.94873046875, 0.1212158203125, 1.4140625, 0.982421875, -0.09393310546875, 0.1546630859375, 0.2281494140625, 0.73095703125, 0.44189453125, 0.27783203125, 0.09710693359375, -0.08612060546875, -1.330078125, -0.79833984375, -0.1192626953125, 0.2308349609375, 0.56103515625, -1.8486328125, -0.57373046875, 0.54150390625, 1.046875, 0.6552734375, 0.31787109375, -1.2490234375, 0.5146484375, 0.1251220703125, -0.216552734375, 0.509765625, 0.95703125, -1.2041015625, -0.03338623046875, 0.6572265625, -1.94921875, -0.3408203125, 1.587890625, -0.7265625, -0.233642578125, -0.12347412109375, -1.9609375, -0.8369140625, -0.87548828125, 0.6650390625, -3.001953125, -0.74658203125, -1.08984375, 0.4130859375, 0.568359375, 0.1885986328125, 0.6337890625, -0.310791015625, 1.611328125, -0.7705078125, -0.289794921875, 0.7353515625, 0.79541015625, -0.986328125, -0.446044921875, -0.0274658203125, -0.8779296875, 1.4052734375, 2.001953125, -0.1263427734375, -2.162109375, 0.1317138671875, 0.455078125, -0.65234375, 0.65185546875, 1.3291015625, -1.234375, -0.3203125, -1.2431640625, -0.2025146484375, 0.5205078125, 2.369140625, 0.005619049072265625, -1.4228515625, -1.5849609375, 0.53955078125, -0.343017578125, -0.286865234375, -0.148681640625, -0.681640625, -0.46923828125, -0.279541015625, -0.4228515625, 0.1796875, -1.4951171875, -0.468017578125, -0.432861328125, 2.236328125, 0.259765625, 0.0100250244140625, 1.388671875, 0.4091796875, -0.31640625, 1.0068359375, 0.1815185546875, -0.15185546875, -1.3583984375, -0.302001953125, 0.6796875, 1.509765625, -0.323974609375, 0.6455078125, 0.17578125, 0.78955078125, 0.62939453125, -2.396484375, -0.1480712890625, -1.7158203125, -0.625, 1.333984375, 0.638671875, -0.1365966796875, -1.232421875, 1.7529296875, 0.77587890625, 2.365234375, 0.74560546875, 0.1982421875, -1.13671875, -0.22412109375, 0.10223388671875, -1.1650390625, 0.92822265625, 1.0068359375, 0.12744140625, 1.9130859375, 0.958984375, 0.9599609375, -1.6220703125, 0.2332763671875, -0.6650390625, -0.376220703125, -0.34423828125, 0.90869140625, -1.4365234375, -0.53125, -0.787109375, 0.8359375, 2.69140625, 0.3681640625, 1.9169921875, -0.7978515625, 0.2412109375, 0.67138671875, 0.8974609375, 0.556640625, -0.143310546875, 0.8466796875, 0.345458984375, 0.64208984375, 0.0173492431640625, -0.3427734375, -0.87646484375, -0.56689453125, -0.51220703125, -1.1220703125, 1.1845703125, -0.115966796875, -0.884765625, 1.01953125, -0.1748046875, -0.5458984375, -0.1414794921875, 0.040679931640625, -0.57373046875, -0.65771484375, -1.3232421875, -0.106201171875, -0.9189453125, -0.407470703125, -1.6552734375, -1.3740234375, -0.5849609375, -0.8232421875, -0.90966796875, 0.80712890625, 0.11126708984375, 2.177734375, 0.1331787109375, 0.87353515625, -0.269775390625, 0.34912109375, 1.0224609375, 0.94189453125, -1.703125, 0.46533203125, 0.353515625, -1.7861328125, 0.46533203125, -0.6875, -0.89404296875, -0.90576171875, -0.1556396484375, -1.1259765625, 1.4072265625, 0.357666015625, 0.7333984375, -1.0849609375, -0.89501953125, -0.62060546875, 1.115234375, 0.36962890625, -0.3203125, 0.373046875, 0.31103515625, -0.07574462890625, -0.07611083984375, 0.479736328125, -0.1697998046875, -1.2119140625, -0.451171875, 1.3310546875, 0.349853515625, 0.484130859375, -1.2509765625, -0.85107421875, -2.39453125, 1.4384765625, -1.66796875, -0.79541015625, 0.08331298828125, -0.9052734375, -1.0087890625, -0.8408203125, -1.5546875, -0.9326171875, 2.44921875, -0.289794921875, -1.0625, -1.4228515625, 0.2347412109375, 0.036163330078125, -1.8125, 0.564453125, 1.013671875, -1.3466796875, -0.1751708984375, 0.66357421875, -1.2939453125, -0.310791015625, -0.60546875, -1.7177734375, 1.802734375, -1.3037109375, 0.9697265625, -0.0423583984375, -1.1064453125, -1.216796875, 1.783203125, 0.09503173828125, 1.6171875, -0.127685546875, -0.65185546875, 0.443115234375, 0.82568359375, 0.078369140625, -1.0693359375, 0.52490234375, -0.76513671875, 0.98486328125, 0.27099609375, 0.9404296875, -1.3984375, 0.76416015625, -0.67333984375, -0.541015625, 0.201416015625, 0.0531005859375, -0.339111328125, -0.7978515625, -0.99853515625, 1.9921875, 0.298828125, 0.4990234375, -0.422119140625, 1.521484375, -0.1402587890625, 0.1636962890625, 0.75439453125, -0.666015625, -2.18359375, -0.09735107421875, -0.267333984375, -0.4189453125, -2.29296875, 0.72802734375, -0.1141357421875, 0.0921630859375, 0.5908203125, 0.94775390625, -1.310546875, 0.344970703125, 0.50830078125, 0.6318359375, -0.63623046875, -1.9951171875, 0.52294921875, 0.86376953125, 0.01126861572265625, -0.0136566162109375, 1.9921875, -1.1083984375, -0.290771484375, -2.27734375, 1.3330078125, 0.0950927734375, -0.381103515625, 1.287109375, -0.302978515625, -0.72509765625, -0.505859375, 0.57470703125, 0.5576171875, -0.29541015625, 0.99462890625, 0.086181640625, 1.0087890625, -1.125, -0.52978515625, 0.1650390625, -0.284423828125, 0.298583984375, -0.485107421875, -0.75537109375, -0.246826171875, -1.2451171875, 1.5888671875, 0.56591796875, 0.25146484375, 0.716796875, 1.3623046875, -0.3916015625, -0.374755859375, 1.2392578125, 0.1593017578125, -2.1171875, 2.431640625, -0.53564453125, -0.81982421875, 0.36083984375, 0.88037109375, -0.046173095703125, 1.349609375, -1.0048828125, -1.0947265625, -0.28125, -1.0966796875, 0.3955078125, 1.4892578125, -0.5673828125, 1.1123046875, -0.853515625, -1.1826171875, 1.1240234375, 1.75, -0.343505859375, 0.9658203125, 0.203125, 0.320068359375, -0.37158203125, 0.00861358642578125, 0.79296875, 0.51220703125, -0.2841796875, 1.8974609375, 0.324951171875, 0.619140625, -0.34521484375, 1.517578125, 0.75341796875, -0.0440673828125, -1.1171875, -1.703125, -0.6162109375, -1.4716796875, -0.71826171875, -0.8984375, -0.414794921875, 0.60009765625, -0.154296875, -0.392578125, -0.006847381591796875, 0.453857421875, -0.311279296875, 0.97607421875, -0.51806640625, -0.7333984375, 0.7939453125, -0.1949462890625, -1.3603515625, 0.22509765625, 0.86376953125, 0.9287109375, 0.466552734375, 0.388671875, 0.98779296875, -0.99365234375, 2.0390625, 1.2548828125, 0.54248046875, -1.8896484375, 1.96484375, -1.38671875, 0.798828125, 0.73876953125, -0.3828125, 0.406982421875, 0.369384765625, -0.896484375, 0.49951171875, -0.8232421875, 0.481689453125, -0.91162109375, 0.576171875, -2.4140625, -0.357666015625, -0.791015625, -0.73876953125, -0.68017578125, -0.55029296875, -0.028076171875, -0.15185546875, -0.72607421875, 0.16650390625, 0.87255859375, 0.609375, 0.05841064453125, 1.7314453125, -0.312255859375, 0.2252197265625, 0.087158203125, -0.7666015625, 0.91650390625, -0.697265625, 1.28515625, -1.0419921875, 0.42626953125, -0.480712890625, -1.2255859375, 1.271484375, -0.1690673828125, -0.180419921875, 0.210693359375, 1.015625, 0.385986328125, -0.21728515625, -1.0537109375, -0.339111328125, 0.0687255859375, 0.3203125, -0.24462890625, -1.5478515625, 0.07080078125, 0.99658203125, -0.323486328125, -1.9482421875, -0.35302734375, 0.05517578125, 2.15234375, -1.2255859375, -1.2197265625, 0.51220703125, 0.63525390625, 1.7099609375, -0.43408203125, -0.124267578125, -0.338134765625, -0.18310546875, 1.52734375, -0.8623046875, -1.138671875, -0.62109375, 0.859375, -0.8056640625, -0.66357421875, 0.291015625, -0.578125, -1.134765625, 0.2257080078125, -1.55078125, 1.25, -1.1220703125, 0.66015625, -0.43798828125, -0.25390625, 0.299560546875, -0.5771484375, 1.0166015625, 0.10577392578125, -1.09765625, -0.225341796875, 0.6875, -0.07513427734375, -1.49609375, -1.08984375, 1.28125, -0.78369140625, -0.0987548828125, -0.53564453125, -1.107421875, 0.755859375, -0.69775390625, -0.5302734375, 0.9248046875, 0.73583984375, 0.70654296875, 0.38427734375, 2.02734375, 0.77880859375, -0.408935546875, 1.228515625, -0.7919921875, 1.0, -1.0625, 0.294921875, 1.2236328125, -1.5009765625, 0.412841796875, 1.34765625, -0.2186279296875, 1.181640625, 0.2091064453125, -1.958984375, 0.2783203125, -0.8671875, 0.66015625, -0.1328125, -0.6591796875, -1.1650390625, -0.529296875, -0.2459716796875, 0.131103515625, -0.244140625, -0.278564453125, 0.240234375, 0.865234375, -1.4453125, -0.3203125, 0.41455078125, 0.9140625, -0.2340087890625, -0.10040283203125, -0.595703125, 1.23828125, -1.916015625, 0.316650390625, 0.2412109375, 1.1474609375, 0.30908203125, -0.0235137939453125, -0.259765625, 0.76611328125, -0.6015625, 1.2216796875, -1.294921875, -1.025390625, 2.6796875, 0.2255859375, 1.779296875, -1.5439453125, -0.318115234375, -1.2275390625, 0.432373046875, 0.302490234375, -1.4775390625, -0.67431640625, 0.385986328125, 1.447265625, 1.1240234375, -0.056640625, -0.07220458984375, 1.2265625, -1.2412109375, 0.67236328125, -0.2471923828125, -1.4462890625, -0.7724609375, 0.65478515625, 0.93408203125, -0.2237548828125, -0.5361328125, 1.3056640625, 0.216552734375, 0.2354736328125, -0.316650390625, 1.4013671875, 0.32080078125, 0.744140625, 0.270751953125, -1.685546875, -0.8056640625, 0.37060546875, -0.1011962890625, -1.115234375, 0.2413330078125, -1.322265625, -1.416015625, 0.471923828125, 0.89404296875, -1.1455078125, -0.52001953125, 0.2308349609375, -0.1617431640625, -2.02734375, 1.419921875, -0.29541015625, 0.3486328125, 0.21826171875, 0.90185546875, 0.2193603515625, -1.3017578125, 1.0185546875, 0.48583984375, -0.255615234375, -0.474853515625, 0.041595458984375, -1.583984375, 0.71826171875, 1.984375, 0.27392578125, 1.5517578125, 1.392578125, 0.9150390625, 0.75146484375, 0.212646484375, 1.5947265625, 0.178466796875, -0.1090087890625, -0.61181640625, 0.004222869873046875, -1.744140625, 0.10980224609375, -1.3935546875, 1.154296875, 0.421630859375, -1.1767578125, -1.201171875, 0.87939453125, -0.99853515625, -0.72607421875, -1.0654296875, 0.59814453125, 1.4111328125, -0.44580078125, 2.15625, 1.6982421875, -0.92724609375, 1.5048828125, -0.402587890625, 1.0478515625, -0.5869140625, -1.111328125, 0.187255859375, 0.56640625, -0.90625, 1.865234375, 1.1484375, 0.27099609375, 1.142578125, -0.03216552734375, -0.6572265625, 0.25634765625, -1.970703125, -0.75390625, -2.62109375, 0.841796875, -0.230224609375, 0.142333984375, -0.67333984375, 0.85546875, 0.31103515625, -1.787109375, 0.31591796875, 1.41796875, 0.0870361328125, -0.08404541015625, 0.446533203125, 0.75927734375, -1.478515625, 0.372314453125, -0.2802734375, 1.4853515625, -2.38671875, -0.268310546875, -0.2318115234375, 1.2060546875, -1.419921875, 0.9990234375, -1.1689453125, 0.837890625, -0.83740234375, 0.327880859375, -0.8466796875, -0.56005859375, 0.462890625, 0.84326171875, 2.001953125, 0.237548828125, 0.57080078125, -0.296630859375, -0.58740234375, -0.272216796875, -0.454345703125, -1.6171875, 1.2646484375, -0.88525390625, -1.560546875, 1.2451171875, 0.454345703125, 0.418212890625, 0.003353118896484375, 0.76025390625, -1.3681640625, 1.08203125, -0.2685546875, 1.041015625, -1.0283203125, 0.308837890625, 0.32958984375, 0.46630859375, 0.2347412109375, 1.0400390625, 0.40625, -1.158203125, 0.156494140625, 0.351318359375, -0.17138671875, 0.30517578125, 1.4111328125, 0.6298828125, 0.1920166015625, -0.326904296875, -0.5341796875, -1.0927734375, -0.0262298583984375, 0.33935546875, -0.0188446044921875, 0.07147216796875, -0.0948486328125, -0.032135009765625, 2.1015625, 0.9033203125, -1.5380859375, 1.09375, 0.28369140625, -0.1337890625, 1.0087890625, 0.7265625, 1.2490234375, -0.21826171875, -0.122314453125, -0.308837890625, 0.802734375, -0.00341033935546875, 0.0217742919921875, 0.88037109375, -0.78271484375, 0.58740234375, -0.26513671875, 0.353515625, 1.1630859375, -0.440673828125, 1.21484375, -0.6376953125, 1.8095703125, -0.234619140625, -0.465576171875, 1.6181640625, 0.193603515625, -0.0968017578125, -0.71044921875, -1.037109375, -1.7373046875, 0.560546875, -0.37939453125, 0.798828125, 0.9375, 0.51904296875, -0.03656005859375, -1.197265625, 1.5400390625, 0.60498046875, 0.58740234375, -0.77392578125, -0.4697265625, -1.5849609375, 0.046539306640625, 0.411376953125, -0.8486328125, -0.576171875, 0.9580078125, -1.318359375, -2.146484375, -1.8876953125, 0.0171661376953125, 0.9482421875, 1.2490234375, -0.178466796875, -0.3818359375, -0.09967041015625, 1.15234375, 1.4892578125, -1.5263671875, -0.578125, 0.380615234375, 0.1837158203125, 1.6044921875, -0.07879638671875, 0.11468505859375, 1.197265625, -0.254638671875, -1.4755859375, 0.08001708984375, -0.60546875, -0.480712890625, 1.0517578125, -1.080078125, 0.125244140625, 0.468505859375, -1.2373046875, -2.69921875, -1.3740234375, 1.578125, 0.09674072265625, 1.3525390625, -0.28271484375, -1.7529296875, -0.1036376953125, -0.78173828125, 0.94140625, -1.28515625, -0.388427734375, 1.060546875, 0.66015625, 0.91162109375, -1.5439453125, 0.595703125, -1.646484375, 0.2103271484375, -0.27392578125, -0.345947265625, -0.070068359375, 1.7431640625, 0.32666015625, -0.697265625, 1.8359375, -0.478759765625, 1.9423828125, 0.5830078125, 0.6708984375, -0.4697265625, -1.0166015625, 1.0458984375, 0.9033203125, 1.640625, 0.51318359375, 1.8212890625, 0.7470703125, -1.02734375, 1.2021484375, -1.197265625, -0.5390625, -0.2218017578125, 0.486083984375, -0.63427734375, 0.57470703125, -1.189453125, -0.08514404296875, -1.208984375, -1.416015625, -0.765625, -1.4287109375, 0.15771484375, -1.1123046875, 0.344482421875, -0.0450439453125, -0.200439453125, -0.609375, 0.4423828125, -0.319091796875, 0.425537109375, -0.254150390625, -1.24609375, 0.39208984375, -0.826171875, 0.2354736328125, 1.28515625, 1.9365234375, -1.2666015625, -1.859375, -0.0012302398681640625, 0.12213134765625, 0.51416015625, -0.5498046875, -0.96826171875, 1.220703125, 0.352294921875, -0.415771484375, -0.79345703125, -0.40673828125, -0.3857421875, -0.409912109375, -0.4521484375, 0.35546875, 1.0029296875, 0.0355224609375, -0.11553955078125, -1.5380859375, 1.3388671875, 0.046966552734375, -1.681640625, -0.568359375, -1.021484375, -0.2587890625, -0.5732421875, -0.93310546875, 1.099609375, 0.94921875, -0.9228515625, -1.208984375, 0.400146484375, 0.6875, -0.0316162109375, -0.445556640625, -1.8076171875, -1.1767578125, 1.33203125, 0.2340087890625, -0.72412109375, 0.10760498046875, -0.458984375, 0.1776123046875, -1.4248046875, 0.8486328125, 0.77587890625, -1.0341796875, 0.50732421875, -1.763671875, -0.0231170654296875, 0.037017822265625, -1.7734375, -1.0517578125, -0.84912109375, -0.482177734375, -1.3046875, 0.2227783203125, -1.486328125, 1.3935546875, 0.099609375, -0.533203125, -1.00390625, -0.10797119140625, -1.984375, 0.9169921875, 1.6279296875, 0.44921875, 0.3095703125, 1.1181640625, 1.5498046875, 1.083984375, -0.442626953125, -0.47509765625, -0.8955078125, -0.470947265625, 0.295166015625, -2.171875, 0.36572265625, -0.028289794921875, -1.283203125, -1.89453125, -0.07440185546875, 2.376953125, -1.3173828125, 0.472412109375, 0.1396484375, -0.2626953125, -0.32275390625, 1.1005859375, 1.7099609375, -1.4091796875, 0.0802001953125, -0.62744140625, -0.53564453125, -0.1983642578125, -1.177734375, 1.88671875, -0.84521484375, 0.42431640625, 1.453125, -0.87646484375, -0.4912109375, 1.32421875, -0.2900390625, 1.0771484375, -0.59130859375, 0.02166748046875, -0.43798828125, -2.203125, 0.65478515625, -0.479248046875, 0.61181640625, 1.3779296875, -1.3603515625, -0.315673828125, -0.7080078125, -0.8408203125, 1.3076171875, 0.45166015625, -0.61865234375, -0.2783203125, -0.0762939453125, 1.3984375, -1.0927734375, 0.1385498046875, 0.03094482421875, 0.462890625, -0.6455078125, -1.384765625, 0.51904296875, -2.16796875, 1.865234375, -1.01171875, -2.1875, 1.9443359375, 0.035003662109375, 0.970703125, 0.56591796875, -0.1988525390625, 1.681640625, 1.765625, 0.0694580078125, -0.77099609375, 1.2822265625, -0.3828125, 0.2091064453125, 0.1837158203125, -0.81494140625, -1.38671875, -0.08843994140625, 0.6806640625, 0.38916015625, -0.6875, -2.576171875, 1.1015625, 2.158203125, 0.357177734375, -0.56982421875, 0.7861328125, 1.849609375, 1.5205078125, -0.67919921875, -0.039154052734375, -0.45458984375, -0.96875, 0.447021484375, -0.01317596435546875, -0.59033203125, -0.158203125, -0.63330078125, -0.50537109375, 0.1656494140625, 0.98681640625, -0.94580078125, 0.548828125, 3.5859375, 1.6796875, 0.0029811859130859375, 0.101318359375, -1.080078125, 1.9931640625, -0.137939453125, 0.77880859375, 1.341796875, -2.609375, 0.455810546875, 0.611328125, 2.365234375, -1.73828125, 1.0654296875, -0.5888671875, 0.1019287109375, 0.6083984375, 0.796875, 0.828125, -0.190185546875, -1.7587890625, 0.54345703125, -0.472900390625, 2.3984375, -0.07550048828125, 1.93359375, 0.12481689453125, -0.65625, 0.299072265625, 2.337890625, -0.0438232421875, 1.62890625, -0.7998046875, 0.034332275390625, 0.216064453125, 1.7626953125, -0.56591796875, 0.457275390625, 2.1953125, 0.7548828125, 0.1259765625, 0.18603515625, -0.485107421875, -0.46044921875, 0.65478515625, -1.2529296875, -0.0770263671875, 0.43408203125, 1.599609375, 1.16796875, 1.0537109375, 1.6259765625, -1.2470703125, -2.0234375, 0.78369140625, 0.37744140625, 1.576171875, -1.1533203125, -0.279541015625, -0.1143798828125, 0.451171875, -0.36376953125, -0.1815185546875, 0.712890625, 0.334716796875, -0.2783203125, -1.76953125, -0.794921875, 0.52734375, -0.11151123046875, -0.517578125, 0.55859375, -0.3359375, 1.673828125, -0.33984375, -0.276123046875, -0.0282440185546875, -0.693359375, 0.33935546875, -0.66015625, 1.4765625, -0.79150390625, -1.59375, 0.7001953125, -1.810546875, 1.662109375, -0.252197265625, -0.304443359375, 0.2061767578125, -1.3662109375, -0.67578125, -0.478271484375, -2.576171875, -1.263671875, 1.8994140625, 1.6806640625, -0.77099609375, -1.0302734375, -1.341796875, -1.0458984375, 0.716796875, -0.1416015625, -0.2705078125, -0.050018310546875, -0.98974609375, 1.1298828125, 2.083984375, 1.7412109375, -0.440185546875, 0.1776123046875, -0.317138671875, 1.0234375, 0.1353759765625, 0.431884765625, -1.443359375, 0.260986328125, -0.490478515625, -1.5068359375, -0.568359375, -0.05950927734375, -1.2119140625, 1.2705078125, 1.251953125, -0.61279296875, -0.55859375, 1.1796875, 1.4306640625, 1.1328125, -0.556640625, -1.3642578125, 0.36669921875, 0.07794189453125, 0.312255859375, 1.1337890625, -0.2548828125, 0.685546875, 1.4931640625, -0.7822265625, -0.154296875, 2.431640625, -0.7255859375, -0.70556640625, 1.453125, 0.293701171875, 0.386474609375, 0.6259765625, -0.7822265625, 0.64990234375, 1.8349609375, -1.302734375, -0.5205078125, -1.6943359375, -0.1376953125, -0.35595703125, -0.184814453125, -1.0595703125, -0.97412109375, 0.91845703125, -0.30419921875, 0.8037109375, 1.12890625, 0.5, 1.3125, 2.2421875, 0.313720703125, -0.52685546875, -1.2568359375, 0.83251953125, 1.146484375, 0.450439453125, -1.029296875, -0.445556640625, -0.900390625, 0.290283203125, 2.427734375, -0.0557861328125, -0.79052734375, 1.1630859375, -0.71142578125, -0.59814453125, -0.191650390625, -1.8271484375, -2.44140625, 0.0024852752685546875, -1.8330078125, -0.87939453125, -0.669921875, -0.16796875, -1.037109375, 0.4384765625, -0.6240234375, 1.2314453125, -0.0274810791015625, 1.580078125, -0.359130859375, -0.85888671875, 0.8515625, 0.111572265625, -1.2626953125, 1.8447265625, -0.396728515625, -0.44921875, 1.1083984375, 1.0185546875, 0.337646484375, -0.4453125, 0.69189453125, 0.79541015625, -0.1572265625, -1.1474609375, 0.434326171875, 0.86083984375, 0.0927734375, -0.92919921875, -0.2119140625, 0.1312255859375, -1.1552734375, -0.9404296875, -0.406005859375, 0.390625, 1.029296875, -1.5947265625, -0.517578125, 0.138916015625, 0.38916015625, 0.01242828369140625, 1.1474609375, -0.77978515625, 1.546875, 0.55615234375, -0.9365234375, -0.61962890625, -0.218994140625, 0.9296875, -0.99609375, -0.98779296875, -1.310546875, -0.415283203125, 0.41064453125, -0.3837890625, 1.1630859375, -0.28759765625, 2.3359375, 2.654296875, 0.21484375, -0.301025390625, -0.03271484375, 0.2001953125, 1.0947265625, 0.266845703125, 0.591796875, -0.08270263671875, -0.467529296875, -0.03271484375, 0.94091796875, 1.7861328125, 1.9404296875, -0.394287109375, -0.8310546875, 1.5703125, 0.2117919921875, 1.353515625, -0.46484375, -0.57958984375, 0.1663818359375, -0.1082763671875, 0.52734375, -0.63427734375, -0.15087890625, -1.41015625, 1.5498046875, -1.0361328125, 0.85009765625, -1.2783203125, 0.95166015625, 1.072265625, 0.51513671875, 0.9677734375, 2.146484375, -1.4765625, 0.33984375, 0.720703125, -0.9873046875, -0.291259765625, -1.015625, -0.97314453125, 0.68408203125, 1.1044921875, -0.85888671875, -0.2447509765625, -0.8798828125, -0.29638671875, -1.1162109375, 1.94140625, -1.59765625, -0.9833984375, 1.32421875, 1.587890625, -1.4453125, -1.365234375, 0.0513916015625, 2.1640625, 1.8017578125, -0.779296875, -0.65283203125, 1.5361328125, -0.38818359375, -0.76171875, -2.1015625, 0.9521484375, 0.86279296875, 0.240478515625, -1.896484375, -1.9267578125, -0.8310546875, -0.81884765625, -0.59326171875, -1.9326171875, 0.89892578125, 1.5849609375, 0.595703125, 0.310791015625, -0.35986328125, 0.68798828125, 0.337158203125, -0.580078125, -0.451171875, 0.5361328125, -1.4326171875, 0.263916015625, 0.47900390625, -0.2049560546875, -0.291015625, 1.3310546875, -0.4365234375, 0.634765625, -1.4140625, -1.0927734375, 0.1488037109375, -0.84326171875, 0.481201171875, 0.59033203125, 0.0246429443359375, 0.029632568359375, -1.130859375, 1.9541015625, 1.4716796875, -0.53515625, 1.0185546875, 1.0263671875, -0.72705078125, 0.2298583984375, -0.72900390625, -0.5673828125, 1.791015625, 0.6474609375, -0.337646484375, -1.0703125, 1.044921875, 1.0791015625, 1.470703125, -1.1552734375, 1.0380859375, -1.8408203125, 0.595703125, 0.485595703125, -2.001953125, 0.484130859375, 1.3134765625, 0.0116729736328125, 0.35107421875, 0.184326171875, 0.242919921875, 0.2384033203125, -1.310546875, 1.181640625, -0.65869140625, 0.3486328125, -1.6435546875, 1.2607421875, 0.4990234375, -1.2294921875, -0.6259765625, -0.2283935546875, 1.1142578125, 1.556640625, -0.306396484375, 1.0400390625, -0.12396240234375, -0.323974609375, 1.6376953125, -0.849609375, 0.210693359375, 0.71435546875, 0.70263671875, -0.6982421875, 1.904296875, 1.2802734375, 2.2890625, 1.326171875, -0.83447265625, 0.375732421875, -1.576171875, -0.97900390625, -0.09259033203125, -0.454833984375, 2.306640625, 3.224609375, -0.34521484375, 1.224609375, -1.3916015625, 0.324462890625, -0.544921875, 0.06378173828125, 0.461181640625, -2.068359375, -0.9873046875, -0.68115234375, -1.0654296875, -0.830078125, 0.5107421875, -0.99462890625, 1.2646484375, -1.1416015625, 0.92333984375, -1.6474609375, 1.0087890625, 1.20703125, 0.169189453125, 0.69580078125, -0.8193359375, -1.2021484375, 1.28515625, -1.076171875, 2.3203125, 0.366943359375, 0.00748443603515625, 0.2069091796875, 0.08746337890625, -0.806640625, 0.986328125, -0.4658203125, 1.1796875, 0.5361328125, -2.431640625, 1.0048828125, -0.0802001953125, -0.72412109375, 0.95166015625, -1.091796875, -1.0380859375, 0.489990234375, -0.07061767578125, -0.58984375, -0.0240478515625, 0.7431640625, -0.28955078125, 0.673828125, -1.697265625, 1.30859375, 1.4541015625, 0.339599609375, -0.437744140625, -0.43505859375, 0.5322265625, 0.5908203125, 0.51416015625, 1.302734375, 0.55029296875, 0.441162109375, -0.38671875, 0.415771484375, 0.402099609375, 1.064453125, 0.71728515625, -2.1328125, 0.207275390625, 0.65283203125, 1.4921875, 1.072265625, 0.1641845703125, 0.259765625, -0.904296875, -0.1837158203125, 0.8916015625, 0.72119140625, -1.6748046875, -0.765625, 0.74755859375, 0.297607421875, -0.492431640625, 0.78955078125, -1.373046875, 0.8447265625, 0.45361328125, -1.8271484375, -0.262451171875, -0.367431640625, -2.435546875, -0.3349609375, 0.47998046875, -0.68212890625, -0.0016183853149414062, -1.7744140625, -0.41015625, 0.484375, -2.16015625, 0.491943359375, 0.62939453125, -1.185546875, 0.51611328125, 0.14990234375, 1.53515625, 0.21240234375, 1.3671875, 0.64013671875, 0.67333984375, 0.50439453125, -1.0068359375, 0.3193359375, -1.24609375, -0.4990234375, -1.3232421875, -0.947265625, -1.076171875, -1.126953125, -0.59326171875, 1.048828125, -0.771484375, 0.8515625, 2.232421875, -0.70947265625, -1.7705078125, 0.0364990234375, 0.413818359375, -0.9931640625, 1.037109375, 1.1630859375, 0.473876953125, 1.31640625, 0.348388671875, 0.8720703125, 0.39111328125, -0.923828125, 0.0102081298828125, 0.1951904296875, -0.403564453125, -1.2353515625, 0.537109375, 0.1090087890625, 1.8251953125, -1.83984375, -0.12261962890625, -0.6005859375, 1.205078125, -0.509765625, 0.68212890625, 0.1767578125, 0.07275390625, 1.310546875, -0.328369140625, -2.078125, -0.8095703125, 0.487548828125, 1.7666015625, -0.00949859619140625, 0.06378173828125, -0.35986328125, -0.7265625, -1.857421875, -0.87744140625, -0.22216796875, -1.1064453125, 0.6591796875, -0.1583251953125, 0.7451171875, 0.609375, 0.198974609375, -0.8759765625, -1.4384765625, -2.46484375, -1.1142578125, -0.20947265625, 0.1737060546875, -0.44775390625, -0.297607421875, -1.05078125, 1.072265625, 0.3369140625, 2.02734375, -0.75048828125, 0.552734375, 0.1519775390625, 0.08056640625, 0.0322265625, -0.426025390625, 0.006649017333984375, 0.68310546875, 0.5302734375, 0.248779296875, 2.70703125, 1.056640625, 0.2440185546875, -1.0595703125, 0.40380859375, -0.75, -0.0305938720703125, 0.0853271484375, 0.062744140625, -0.72412109375, 0.90283203125, -1.134765625, 0.44970703125, -1.203125, 0.3310546875, 0.40087890625, -1.0615234375, -2.09375, -1.0498046875, 1.2626953125, -0.62744140625, -1.486328125, -0.0689697265625, -0.048675537109375, 0.31396484375, 2.12109375, -0.04742431640625, 1.2080078125, 0.03143310546875, 0.391845703125, 0.199462890625, 0.8173828125, -1.044921875, 1.1279296875, -0.07379150390625, -0.11224365234375, -0.91943359375, -0.31298828125, 1.068359375, -0.650390625, -0.0498046875, 0.8984375, -0.55859375, -1.3388671875, 0.61865234375, 1.0224609375, 0.2142333984375, -1.072265625, 0.544921875, -1.3681640625, -0.0908203125, -0.521484375, 0.16455078125, -0.2230224609375, 0.10919189453125, -0.490966796875, -0.0960693359375, 0.90576171875, 0.7333984375, 0.583984375, -0.075927734375, 1.0234375, -1.0322265625, 0.64697265625, 0.51171875, 0.080810546875, -0.50634765625, -0.107666015625, -0.27783203125, 2.099609375, -0.0743408203125, -2.33203125, -0.6630859375, 1.9462890625, -1.52734375, 0.2186279296875, 0.84912109375, -0.201416015625, 1.5068359375, 1.150390625, -1.1630859375, -0.298095703125, -0.29638671875, 0.376953125, 0.53076171875, 0.61865234375, 0.2347412109375, 1.1787109375, 0.50537109375, -2.8125, -0.284912109375, 1.3466796875, -0.59521484375, 0.57958984375, -1.1201171875, -1.1484375, -0.68408203125, -0.315673828125, -0.8388671875, -0.256591796875, 1.849609375, 0.71923828125, 0.67431640625, -0.71630859375, 1.4765625, 1.4423828125, 1.837890625, 0.14306640625, -1.09765625, -1.638671875, 0.8212890625, -0.96923828125, 0.42919921875, -0.302734375, 0.5927734375, 0.64697265625, 0.9990234375, -1.080078125, 0.04193115234375, -1.333984375, 0.054351806640625, 0.422119140625, -0.8798828125, -0.66455078125, 0.7626953125, 0.176025390625, 0.019195556640625, 0.373291015625, -0.96044921875, 0.0675048828125, 0.70263671875, 0.469482421875, -1.7265625, -0.4521484375, -0.71533203125, 0.7490234375, -0.796875, -0.6318359375, -0.1370849609375, 0.25390625, -0.8037109375, 0.80224609375, -0.5654296875, -0.87939453125, -0.2412109375, -1.837890625, 0.900390625, 0.312744140625, -0.5625, 0.10687255859375, -0.2052001953125, -0.6025390625, 0.346923828125, 0.61376953125, -1.4912109375, 0.95166015625, -0.0338134765625, 1.1806640625, -1.1806640625, -0.41796875, 1.4072265625, -0.0048675537109375, 0.434326171875, -1.9716796875, 0.77392578125, -1.0927734375, -1.0009765625, 0.51318359375, 0.1396484375, -0.65087890625, -0.62353515625, -0.06597900390625, -0.2939453125, -0.8291015625, -0.252685546875, -0.425048828125, -1.2099609375, -0.71875, 1.9609375, -1.349609375, 1.107421875, 1.7568359375, -0.245849609375, -1.4892578125, 0.10321044921875, 1.4609375, -0.3759765625, 1.478515625, -0.99853515625, -0.6416015625, -0.89306640625, 0.35693359375, -0.95654296875, 0.4580078125, 0.16552734375, 0.78076171875, 1.34375, -0.71044921875, 1.3271484375, -0.90234375, -0.239990234375, 2.640625, 0.4345703125, 0.322998046875, 0.343505859375, 0.82275390625, 0.28369140625, -1.05078125, 1.5546875, -2.873046875, -0.77783203125, 0.6435546875, -1.3359375, 2.544921875, -0.6572265625, 0.5615234375, 0.9638671875, -0.47998046875, 0.401611328125, -0.05859375, 1.7783203125, 0.273193359375, 0.414794921875, -0.35205078125, 0.2093505859375, -0.67041015625, -0.1588134765625, 0.428955078125, -0.00913238525390625, -1.1142578125, -0.625, 0.82958984375, 1.5908203125, 0.48828125, 1.5693359375, 0.044281005859375, 0.51171875, -0.138916015625, -0.9990234375, -1.6669921875, -0.08270263671875, 1.1494140625, -0.1295166015625, 0.018829345703125, 0.12042236328125, -0.62060546875, -2.1015625, 0.301025390625, -1.5009765625, 0.81201171875, -0.7080078125, -1.7958984375, 0.44873046875, 0.4033203125, 0.488525390625, -0.99853515625, -0.28125, -1.568359375, 0.025787353515625, -1.169921875, -3.1328125, -0.76611328125, 0.28271484375, 0.73876953125, -0.9765625, 0.314453125, 0.391357421875, -0.908203125, 0.70361328125, 0.59228515625, -1.16015625, 0.58544921875, -1.5263671875, -0.2763671875, -1.9775390625, -0.138671875, 0.049530029296875, -2.912109375, -0.4189453125, -0.1312255859375, 1.337890625, 0.345458984375, 0.5244140625, 0.10626220703125, -2.28515625, -0.2257080078125, 0.020660400390625, -0.41259765625, -0.5947265625, -0.75830078125, 1.2939453125, -0.06500244140625, -1.3115234375, -1.2412109375, -0.286865234375, 0.60009765625, -0.276123046875, 0.78857421875, -0.861328125, -1.294921875, -0.95751953125, -0.9873046875, -1.572265625, 1.1708984375, -2.154296875, -1.34375, 1.09765625, -0.8173828125, 0.1256103515625, -1.015625, 0.5546875, -1.2802734375, 2.4140625, -1.201171875, 0.748046875, 0.97412109375, 0.0621337890625, -0.0587158203125, 0.81298828125, -0.4736328125, -0.134521484375, 0.50439453125, 2.181640625, 0.38330078125, -2.572265625, 0.489990234375, 0.9013671875, 0.1708984375, 1.111328125, 1.3779296875, -0.1429443359375, 1.8056640625, -1.6708984375, -0.60693359375, 0.244140625, 0.125244140625, 0.09466552734375, 0.79736328125, 0.58349609375, 0.67578125, 0.61328125, 0.91845703125, 1.255859375, -0.76025390625, -1.85546875, -1.341796875, 2.470703125, 0.52490234375, -1.7373046875, 1.279296875, -0.239501953125, -0.08782958984375, 2.048828125, -0.1405029296875, 0.92431640625, 2.099609375, 0.576171875, 0.67919921875, -0.91552734375, 0.5361328125, 1.09765625, -0.487060546875, -0.5625, -0.40673828125, -0.0394287109375, -0.33837890625, -0.1744384765625, -0.37939453125, 0.025482177734375, -1.25, -0.374755859375, 0.323974609375, 0.681640625, 0.537109375, -1.201171875, -0.7998046875, 0.751953125, -0.6728515625, -1.2265625, -1.8359375, -0.12310791015625, 0.1741943359375, 0.6982421875, 1.068359375, 1.2333984375, -1.521484375, 0.389892578125, -0.383056640625, 0.258544921875, -2.08984375, -0.89208984375, -0.87255859375, -2.025390625, -1.130859375, -1.9833984375, -0.93701171875, 0.2454833984375, 0.6572265625, 0.37255859375, 1.146484375, -0.10882568359375, 0.69140625, 1.876953125, 0.346923828125, 1.06640625, -1.107421875, -0.184814453125, -0.30810546875, 0.888671875, -0.2841796875, -0.310791015625, 1.4541015625, -0.394775390625, 0.7529296875, 0.2291259765625, -0.1043701171875, 0.004428863525390625, -0.342041015625, 2.46484375, 0.278564453125, -1.244140625, 0.1715087890625, 0.56689453125, 0.3974609375, 0.0069580078125, -0.07305908203125, -0.409912109375, 0.5107421875, -0.354248046875, -0.363037109375, 2.5390625, 0.326416015625, 0.53515625, 0.486572265625, -0.1094970703125, -1.76171875, -0.481689453125, 0.334716796875, 0.84423828125, -0.892578125, -1.51171875, -1.015625, -0.05731201171875, 2.060546875, -0.609375, 0.1336669921875, 0.76171875, 0.623046875, 1.1083984375, 0.5048828125, -0.36669921875, -1.9697265625, -2.111328125, 1.0986328125, 0.498779296875, -2.486328125, -1.3876953125, 0.666015625, 1.5673828125, 0.462646484375, 1.6943359375, 0.7744140625, -1.494140625, -0.8232421875, 1.16796875, 1.759765625, 1.873046875, 0.81298828125, 0.7275390625, 0.183837890625, -2.134765625, 0.1485595703125, -0.7294921875, -0.728515625, -1.375, -1.68359375, -0.11529541015625, -0.306640625, -1.35546875, 0.06597900390625, -0.71875, -0.10009765625, -1.056640625, 0.94482421875, 0.73876953125, -0.428955078125, 1.35546875, -0.47998046875, -0.88330078125, -0.50439453125, 0.0041961669921875, 1.244140625, -0.1641845703125, 1.330078125, 0.41162109375, 0.2236328125, 0.5908203125, 0.07012939453125, 0.6416015625, 0.103271484375, 0.325927734375, -0.09320068359375, -2.416015625, 0.422119140625, -0.77294921875, 0.2308349609375, 0.2646484375, -0.424560546875, -0.1531982421875, -1.0751953125, -0.4267578125, 1.23828125, 0.458984375, 0.482666015625, -1.0810546875, 0.56982421875, -0.8359375, 1.296875, 1.55078125, -0.3583984375, 0.50048828125, -1.076171875, -0.71142578125, -0.482666015625, 0.5537109375, -0.2117919921875, 0.37939453125, -1.146484375, -0.09521484375, 1.3486328125, -1.4345703125, -0.469482421875, -0.451171875, 0.052886962890625, -1.14453125, -1.404296875, -0.07958984375, 1.2392578125, -0.318115234375, -0.5791015625, 0.424560546875, 0.876953125, 0.274658203125, -0.3671875, -0.228515625, -0.27880859375, -0.99462890625, 0.1983642578125, 0.23876953125, 0.09869384765625, -1.04296875, 1.349609375, -1.4033203125, 0.381591796875, -2.5234375, -0.1148681640625, 0.281494140625, 1.138671875, 0.36865234375, 1.2373046875, -0.748046875, 0.62890625, -0.20947265625, 0.51513671875, -0.6689453125, -1.70703125, -2.177734375, 1.2421875, 1.087890625, 1.44140625, -1.5634765625, -0.908203125, 1.697265625, -1.36328125, 0.587890625, -1.3720703125, -0.06219482421875, -0.2880859375, 0.13818359375, 0.04620361328125, -0.085693359375, 0.89697265625, -0.4892578125, -0.90771484375, -0.49609375, -0.30908203125, 0.6806640625, 0.082763671875, 0.7919921875, -0.359619140625, -0.4453125, -1.357421875, -1.1240234375, -0.3525390625, 0.9248046875, -0.267578125, 0.8134765625, 1.2763671875, -0.300048828125, -1.583984375, 0.484130859375, -0.78759765625, -1.1806640625, -1.6240234375, -1.072265625, 0.369873046875, 0.732421875, -0.12646484375, 0.7490234375, 0.341064453125, 0.51123046875, 1.5517578125, 0.99365234375, -0.1175537109375, -0.8056640625, -0.048614501953125, 0.4755859375, -0.50439453125, 0.1571044921875, -0.0826416015625, -0.2276611328125, 1.2001953125, 0.09478759765625, 0.11553955078125, -1.0341796875, -0.455078125, 0.6767578125, -0.47509765625, 1.16796875, -1.85546875, -0.0908203125, 1.115234375, 0.382080078125, 0.91015625, 3.140625, -0.3310546875, -0.69189453125, 0.290771484375, 0.2020263671875, -0.59814453125, 0.56298828125, -1.23828125, -0.3994140625, -0.017791748046875, -0.1617431640625, -0.9140625, 0.76123046875, 1.728515625, 1.9521484375, -1.2724609375, -0.275634765625, -1.01171875, -0.40185546875, -1.5322265625, 0.88720703125, -1.2412109375, 1.6298828125, 0.481201171875, -0.20703125, -0.494140625, -2.146484375, -0.33935546875, -0.64306640625, -0.80517578125, 0.40380859375, -0.48291015625, 1.6806640625, 0.5537109375, 1.623046875, 0.57861328125, 1.87109375, 1.1015625, -1.4775390625, 0.52001953125, -1.345703125, -1.14453125, -1.1650390625, -1.6162109375, -0.3212890625, -0.576171875, -0.161376953125, -0.39794921875, 1.0478515625, -0.82177734375, -0.361572265625, 0.54736328125, 0.2091064453125, 0.458984375, 0.8359375, -0.488037109375, -0.31689453125, -0.424560546875, -0.82421875, -0.1549072265625, 0.7392578125, -0.76171875, -1.376953125, 1.6806640625, -1.14453125, -0.61669921875, -0.46044921875, -0.478271484375, -0.41015625, 0.01873779296875, -1.6025390625, -0.57421875, -0.1490478515625, -0.8681640625, -1.3408203125, 0.45654296875, 0.46044921875, 0.9052734375, -0.478271484375, 1.5458984375, -0.2060546875, 0.2491455078125, 0.9462890625, 1.068359375, -0.84814453125, -0.7568359375, 0.2265625, 0.83642578125, -0.09088134765625, -0.04754638671875, 1.1044921875, 0.1881103515625, 0.80126953125, 0.55322265625, -0.43408203125, 0.65185546875, -0.273193359375, 0.034912109375, 1.2763671875, -2.75, -1.6669921875, -0.32421875, -0.86279296875, 1.4931640625, 1.0068359375, 0.8544921875, -2.986328125, -0.1617431640625, 1.083984375, -0.2364501953125, -0.63623046875, -1.2412109375, 0.12213134765625, -0.1968994140625, 1.396484375, 0.408935546875, 0.5283203125, 0.59375, -0.81005859375, -0.10882568359375, -0.50048828125, 1.2578125, -1.150390625, -0.69970703125, -0.059814453125, -1.8798828125, -0.5859375, 1.509765625, 0.271240234375, 0.55419921875, 0.70947265625, -0.0648193359375, 0.251220703125, -0.47021484375, -0.83154296875, 0.447998046875, -0.1263427734375, -1.9375, -2.248046875, 0.947265625, -1.20703125, -0.7685546875, -1.9677734375, 0.7412109375, 0.92724609375, 0.556640625, -0.279296875, -0.3896484375, 0.44873046875, -0.54736328125, 0.83056640625, -0.129150390625, -0.94091796875, 0.900390625, 1.1318359375, 0.4638671875, 0.479248046875, 1.072265625, 0.77001953125, -0.73486328125, 0.3798828125, 0.495361328125, 1.4013671875, -0.79833984375, -1.63671875, 1.0087890625, 0.69384765625, -1.140625, -2.798828125, -0.859375, -1.5263671875, 0.94775390625, -0.357666015625, 0.0771484375, -0.1849365234375, 1.1279296875, -0.2083740234375, 0.72265625, -0.77880859375, -0.055938720703125, -0.15771484375, -0.6630859375, 0.297607421875, -1.2197265625, 0.001827239990234375, 0.174072265625, 1.107421875, 0.431396484375, 1.1015625, -0.8203125, 0.8095703125, 0.413818359375, 0.650390625, -0.396728515625, 1.0859375, 0.1646728515625, 0.35693359375, 0.306884765625, 0.335205078125, -2.849609375, 1.90234375, -0.2149658203125, -0.42822265625, -0.99169921875, 0.4599609375, 0.1343994140625, 2.166015625, -0.72998046875, -1.072265625, -0.96044921875, 0.049346923828125, -0.480712890625, 1.1103515625, -0.09124755859375, -1.5849609375, 0.275390625, -0.3037109375, 0.103515625, 1.515625, 0.353271484375, -0.53173828125, 0.857421875, -0.08892822265625, -0.71484375, -0.451171875, 0.6435546875, 0.97998046875, -0.56884765625, 1.2421875, 0.91748046875, -0.88720703125, -0.443603515625, -1.4140625, -0.2410888671875, 0.478515625, 0.20849609375, -0.281982421875, -0.31005859375, 1.1591796875, -0.334228515625, 0.68896484375, -2.31640625, 1.29296875, -1.5029296875, 0.623046875, -0.1650390625, 0.7216796875, 1.59375, -0.984375, -1.5986328125, 0.46142578125, 1.537109375, -0.79345703125, 0.65283203125, -0.1273193359375, -0.55908203125, 0.99560546875, -1.12109375, 0.3955078125, -0.1876220703125, 0.88037109375, 1.818359375, 0.311279296875, -0.348388671875, 0.94775390625, -0.71337890625, -0.1348876953125, -0.96142578125, -0.50244140625, 0.888671875, -0.47900390625, 1.2080078125, -0.125732421875, -0.79541015625, -0.4228515625, -0.62109375, -0.91748046875, 0.08203125, -1.572265625, -1.0380859375, 2.251953125, 1.2509765625, -1.5126953125, -0.25244140625, -0.54248046875, -0.4716796875, -0.39306640625, 1.3349609375, -0.26513671875, 0.423828125, 1.091796875, -1.4384765625, -0.030517578125, 0.0986328125, -0.08746337890625, 0.430908203125, -0.168701171875, 0.279296875, 0.5673828125, 0.0009145736694335938, 0.309814453125, -0.8759765625, -0.494873046875, 0.31689453125, -0.73291015625, 0.9755859375, 0.52392578125, 0.398193359375, 0.11505126953125, -0.29443359375, 0.1107177734375, -0.1239013671875, -1.59375, -0.125244140625, -1.4521484375, 1.458984375, -0.42138671875, -0.061248779296875, -1.8671875, -0.034271240234375, -0.392578125, -0.5966796875, -0.734375, -1.09375, -0.43408203125, -0.740234375, 0.69775390625, -0.8720703125, -0.763671875, -0.861328125, -0.2315673828125, -0.310791015625, -0.84814453125, -1.3466796875, -0.5576171875, -0.336669921875, -2.1875, -1.08203125, -0.87060546875, 0.5361328125, -1.4365234375, -0.6298828125, -1.4345703125, 0.5576171875, -1.205078125, -2.541015625, 0.2393798828125, 1.1357421875, -0.5263671875, 0.88232421875, -0.55078125, -0.6845703125, -0.374755859375, 1.8095703125, -0.83544921875, -0.10369873046875, 0.1702880859375, 0.75, -1.05078125, 1.3564453125, 0.93994140625, -0.9638671875, -1.232421875, 0.4453125, -1.7724609375, 1.080078125, -0.132080078125, -0.8916015625, 0.368408203125, -0.05230712890625, -2.052734375, -0.44921875, 0.9560546875, -0.5732421875, -0.0921630859375, -1.083984375, -1.0947265625, -0.31494140625, 0.00720977783203125, -0.037567138671875, -1.4912109375, 0.3447265625, 1.2509765625, 2.05859375, -0.440673828125, 1.8701171875, -0.15380859375, -0.5126953125, -1.2978515625, -0.20263671875, 0.5087890625, -2.818359375, 0.08135986328125, -1.2099609375, 0.75732421875, -0.1717529296875, 0.06561279296875, -1.1416015625, 1.193359375, -1.080078125, -0.253173828125, -0.150146484375, -0.0487060546875, -0.385986328125, -1.6103515625, 0.41455078125, -0.7587890625, -0.43994140625, 0.919921875, -1.3291015625, 0.422607421875, 0.51708984375, -1.1201171875, 0.3271484375, 0.8759765625, 0.35595703125, 2.2265625, -0.90966796875, -0.50244140625, 0.3212890625, 0.1551513671875, 0.662109375, 1.646484375, 1.0400390625, -0.66748046875, 1.1494140625, -0.33642578125, -0.2335205078125, -0.66796875, -0.1700439453125, 0.92529296875, 0.353515625, -1.0263671875, -0.303466796875, -0.35693359375, -0.298095703125, 0.158203125, -1.1044921875, -0.333984375, -0.7177734375, -1.1748046875, -0.455810546875, -0.181640625, -0.1339111328125, 1.5146484375, 0.2423095703125, 1.3681640625, -0.158447265625, 0.482177734375, -1.115234375, -1.517578125, -0.1680908203125, 0.2724609375, -3.291015625, -1.2685546875, 0.708984375, -0.78076171875, -0.345703125, 0.1912841796875, -0.24072265625, -1.064453125, -1.109375, -1.4150390625, 0.360107421875, -0.92724609375, -0.036041259765625, -0.406005859375, -1.8408203125, -1.7275390625, 1.8017578125, -0.1290283203125, 0.53466796875, 1.3623046875, 0.90185546875, -0.75390625, -0.235595703125, -0.270751953125, 1.7548828125, 0.1351318359375, -1.0283203125, -1.3935546875, -0.2235107421875, -1.0947265625, 2.46484375, 0.271240234375, 0.419189453125, 1.03515625, 1.9033203125, -1.5400390625, 1.4326171875, -0.8203125, 1.10546875, 1.0185546875, -0.755859375, -1.1484375, -1.580078125, -1.3310546875, -0.3212890625, -2.4375, -2.7109375, -0.921875, -0.88818359375, 1.4072265625, -1.2353515625, -0.1627197265625, -0.042572021484375, 0.07257080078125, 1.236328125, -0.426025390625, 0.5390625, 1.5712890625, -0.43310546875, 1.3896484375, -1.5927734375, -0.439208984375, 0.27685546875, -0.129638671875, -0.57958984375, -0.6923828125, -1.7265625, -0.47705078125, 1.900390625, -0.485595703125, -0.387451171875, -0.148193359375, 0.70458984375, 2.14453125, 0.6591796875, -0.94970703125, 1.0263671875, -1.4873046875, 1.06640625, -0.11187744140625, 0.462158203125, 0.11517333984375, -0.4638671875, -0.65966796875, -0.015655517578125, -0.94970703125, 0.51611328125, 1.0712890625, -0.47705078125, -1.5361328125, -1.59765625, 1.2138671875, 0.320556640625, 0.08099365234375, 0.705078125, -0.98828125, 0.8876953125, -0.150146484375, 2.375, 0.583984375, 1.080078125, 2.00390625, 0.18359375, 1.7060546875, -1.0859375, 0.041961669921875, -0.450927734375, -0.462646484375, 1.59375, 0.263916015625, 1.0595703125, -0.8798828125, 1.1005859375, 1.9775390625, -2.849609375, -0.01229095458984375, -0.279052734375, -0.26708984375, -0.212890625, 1.322265625, 0.24365234375, 0.1014404296875, 0.262939453125, 0.042083740234375, 0.1497802734375, -0.87841796875, 0.19482421875, -1.421875, -1.6015625, -0.203369140625, -1.9228515625, -0.356201171875, -0.98681640625, 1.1240234375, -0.312255859375, -1.1162109375, 0.057708740234375, 0.77587890625, -0.61083984375, -0.67041015625, 1.439453125, -1.3642578125, 0.304931640625, -0.44140625, 0.556640625, -0.58447265625, 1.009765625, 0.09466552734375, -0.83251953125, -0.62890625, -1.0048828125, 0.279296875, 0.76806640625, 1.2216796875, -1.142578125, -1.6259765625, 1.5771484375, -0.01094818115234375, -0.68212890625, -1.26953125, 0.84521484375, 0.414794921875, 1.6640625, 0.295166015625, -0.11285400390625, -0.41943359375, -0.92041015625, 1.1181640625, -0.10919189453125, 0.7529296875, 0.326904296875, -0.87353515625, 1.0107421875, 1.0322265625, 0.9765625, 0.087890625, 0.1937255859375, 0.2493896484375, -1.2841796875, -0.78173828125, 0.78076171875, -2.22265625, -0.7841796875, 0.68701171875, -0.08612060546875, 0.388916015625, -1.6123046875, 0.36083984375, 1.361328125, -0.91845703125, -0.95068359375, -0.392822265625, 0.34326171875, -0.60986328125, -2.013671875, -0.449462890625, -0.6806640625, -0.455078125, 0.75, 0.00640106201171875, -0.11572265625, 0.294677734375, 0.43310546875, -0.450439453125, 1.00390625, -1.9130859375, 0.5107421875, -0.9619140625, 0.52197265625, 1.8017578125, -0.51708984375, 0.55322265625, 0.79150390625, 1.935546875, 0.2384033203125, -0.2763671875, -0.80908203125, -0.08880615234375, -2.5078125, -0.07672119140625, -0.3203125, -1.37109375, 0.181884765625, -0.45751953125, 0.77880859375, -1.3681640625, -0.91845703125, 0.638671875, 0.1866455078125, -0.20166015625, 1.0390625, -0.92138671875, 0.70458984375, -0.55517578125, -0.3310546875, 0.262451171875, 0.1280517578125, -0.64404296875, 1.9765625, -0.25390625, 0.51806640625, 0.1517333984375, 0.77197265625, -0.4853515625, 1.13671875, 0.376220703125, 0.63037109375, 0.767578125, -0.0008149147033691406, -0.02203369140625, 0.1710205078125, 1.7607421875, 0.97998046875, -0.04351806640625, 0.85888671875, -0.97265625, 0.48095703125, -0.163330078125, 0.904296875, -0.62744140625, -0.2305908203125, -0.65625, 0.485595703125, -0.391357421875, 0.142333984375, -1.7275390625, 0.08221435546875, 1.2451171875, -0.098388671875, -0.274169921875, 0.387451171875, -0.78466796875, -0.58935546875, -0.8466796875, 1.6328125, -1.7841796875, -0.96875, 1.462890625, -1.4111328125, -0.32373046875, 0.08221435546875, 0.463623046875, 0.92919921875, 1.3466796875, -0.49560546875, 1.4677734375, 1.51171875, -0.05859375, -1.0712890625, 0.377197265625, 0.744140625, -0.0095367431640625, 1.333984375, 0.052154541015625, 0.430419921875, 1.0302734375, -0.56689453125, -1.013671875, 0.609375, 0.0677490234375, 0.47119140625, -0.841796875, 0.51416015625, 0.951171875, -0.295654296875, -0.196533203125, -0.3603515625, 0.74755859375, -2.794921875, 0.16162109375, -0.402587890625, 0.86181640625, -1.478515625, -2.201171875, -1.1923828125, 0.276123046875, -0.62109375, -0.38916015625, 0.07666015625, 1.5537109375, 0.94140625, 0.22509765625, -0.288330078125, 2.3671875, 0.8134765625, 0.70263671875, -0.60498046875, -0.07464599609375, -0.224365234375, 0.6298828125, 0.007305145263671875, -0.26904296875, -0.58642578125, 0.1944580078125, 0.56396484375, -0.303466796875, -0.14208984375, -0.1986083984375, -0.77880859375, 0.8310546875, -0.430908203125, -1.134765625, 2.111328125, 0.9404296875, -0.9375, 0.28125, 0.53662109375, 0.0221710205078125, -1.1630859375, 1.9423828125, 0.2978515625, 0.9306640625, -0.7626953125, 1.3837890625, 0.115234375, 1.89453125, -1.548828125, -1.548828125, -0.2425537109375, 1.931640625, -1.2001953125, -1.2216796875, 0.623046875, 0.57470703125, -0.291259765625, 0.28955078125, 0.78662109375, 0.9384765625, -0.58935546875, 0.7421875, -2.01953125, -0.56689453125, -1.228515625, -1.7021484375, 0.1375732421875, -0.08135986328125, -0.08087158203125, 0.1917724609375, 0.0731201171875, 1.7939453125, 0.85400390625, -1.5390625, 0.79931640625, -1.0234375, 0.12030029296875, 0.421875, -0.040557861328125, -0.026275634765625, -1.16015625, -0.3564453125, 0.52734375, -1.5966796875, 0.39990234375, 0.490966796875, 0.9462890625, -0.09613037109375, 1.0703125, 0.7529296875, 0.92578125, 0.69873046875, -0.97314453125, 1.7255859375, -0.251220703125, 1.2724609375, 0.70947265625, 0.100341796875, 1.1943359375, 0.93896484375, 1.2666015625, 1.4453125, -0.06414794921875, -0.66259765625, -0.70263671875, -0.051116943359375, 1.763671875, 1.7353515625, 0.271728515625, 0.40234375, -1.828125, 0.180908203125, -0.8095703125, 0.2423095703125, -0.0052947998046875, 0.58740234375, -0.58447265625, 1.4912109375, 0.050262451171875, 0.30078125, 0.38232421875, -1.9287109375, -0.7705078125, 0.37255859375, 0.24755859375, 1.392578125, 0.94873046875, 1.87890625, -0.634765625, 0.74658203125, 0.077392578125, -0.5595703125, -0.319580078125, 0.4892578125, 0.03265380859375, 1.669921875, -0.95654296875, 0.30078125, 0.259033203125, 0.73095703125, -0.2193603515625, 0.56884765625, -0.865234375, 0.94921875, -0.365966796875, 0.23779296875, -0.69287109375, 0.533203125, -1.3251953125, 0.0628662109375, 0.8623046875, 0.96240234375, 1.453125, -1.5751953125, 1.09765625, -0.1982421875, -0.64404296875, -0.247802734375, -1.3212890625, 2.251953125, 0.298583984375, -0.58447265625, 0.7421875, -0.369140625, 0.765625, 0.49560546875, -1.0498046875, -0.078369140625, 1.701171875, 0.51611328125, -0.603515625, 0.6298828125, -0.4443359375, -0.79296875, 1.1826171875, -0.92919921875, 0.71826171875, 0.07476806640625, 0.89599609375, -3.22265625, 0.7744140625, -0.251953125, 0.87939453125, -0.77490234375, 2.0, 0.169189453125, 0.0343017578125, 1.0263671875, -0.444580078125, 2.15234375, -1.2177734375, 0.71142578125, 1.46484375, 0.0654296875, 0.97900390625, -1.09375, 0.732421875, -0.366455078125, -0.0751953125, -0.109375, -1.0146484375, -0.009246826171875, 0.09442138671875, -1.396484375, -0.285888671875, 1.1328125, 0.1480712890625, 0.6630859375, -0.294921875, -0.7255859375, -0.0198516845703125, 1.9921875, -0.64990234375, 0.73583984375, 0.439208984375, 0.1546630859375, -0.1280517578125, 0.1619873046875, -0.0753173828125, -1.490234375, -0.62255859375, 1.2255859375, 0.177001953125, -0.4609375, -0.95068359375, 0.054290771484375, 1.3984375, 0.08154296875, -0.1414794921875, 0.71337890625, -0.62841796875, -0.896484375, -0.23388671875, 1.189453125, -0.2305908203125, 0.1500244140625, 1.3486328125, 0.35498046875, 0.33935546875, -0.673828125, -1.29296875, -0.64697265625, -1.65234375, -0.69580078125, 0.08477783203125, 0.55322265625, 0.00902557373046875, -0.2215576171875, 1.8466796875, -0.7431640625, 0.53955078125, 0.470458984375, -0.09234619140625, 0.35107421875, 1.4150390625, -0.6630859375, 0.4423828125, 0.41015625, -0.88330078125, -0.4111328125, 0.76416015625, 0.464111328125, 0.69189453125, 0.9560546875, -1.4638671875, 0.274169921875, 0.7080078125, 0.68896484375, 0.32421875, -1.7998046875, 2.654296875, 0.73388671875, -0.74072265625, -0.3369140625, 1.7314453125, -0.344970703125, 1.1318359375, 1.1025390625, 0.305908203125, -1.193359375, 0.42236328125, 0.370361328125, -1.720703125, 0.38525390625, -0.498046875, 1.1474609375, 0.78173828125, 1.5234375, -0.53759765625, -0.70166015625, 0.410400390625, -0.4677734375, -0.72412109375, -0.376708984375, -1.31640625, -1.6279296875, -0.1849365234375, 0.35595703125, 0.75244140625, -0.381103515625, -1.7333984375, -2.583984375, -0.8212890625, 0.6513671875, 0.5634765625, -0.255126953125, -0.98193359375, 0.52685546875, 0.56884765625, 1.20703125, -0.279296875, 0.1273193359375, 1.1015625, -1.3505859375, -1.158203125, 2.59375, -2.109375, -0.462158203125, 0.9736328125, 1.349609375, -1.4541015625, 1.3583984375, -0.155029296875, 1.2763671875, -0.09136962890625, 1.0380859375, -0.68896484375, 0.10626220703125, 1.400390625, -0.0906982421875, -1.5126953125, 0.5068359375, -0.292236328125, 0.1031494140625, -0.99658203125, -0.06427001953125, -2.205078125, 0.0208282470703125, 0.54345703125, 0.5576171875, -0.79345703125, -2.951171875, 0.51416015625, 0.201904296875, 0.135009765625, 0.136962890625, 0.01983642578125, 0.012603759765625, -0.276611328125, -1.5556640625, 0.236083984375, 2.197265625, 0.01467132568359375, -0.49365234375, 0.1851806640625, -0.2171630859375, 0.67529296875, 0.111572265625, 1.927734375, -1.2802734375, -1.0322265625, 0.09271240234375, 0.335693359375, 1.458984375, -0.51171875, -0.0229339599609375, 0.494384765625, 0.0830078125, -0.326171875, -0.60888671875, -0.3486328125, 1.3388671875, 0.2491455078125, -1.0029296875, -0.4775390625, 0.712890625, 0.82861328125, 1.0166015625, -1.3193359375, 2.357421875, 1.69921875, 1.5, -1.7861328125, 0.1077880859375, 2.478515625, -1.40625, -1.5634765625, -0.09747314453125, 1.275390625, -1.44140625, 1.59765625, 1.669921875, 0.548828125, -0.417236328125, -0.5009765625, -0.055999755859375, 1.640625, -0.1583251953125, -0.1451416015625, -0.51806640625, 0.424560546875, 1.509765625, -0.72216796875, -0.69921875, 0.78955078125, 0.07965087890625, 0.35009765625, -0.30517578125, 0.444091796875, 0.55712890625, 1.267578125, 0.865234375, -0.259521484375, 1.158203125, 0.177734375, -0.9296875, 0.310791015625, -1.267578125, 0.32373046875, 0.35400390625, 1.703125, -1.380859375, -0.70068359375, -1.3740234375, -1.974609375, 1.2099609375, -1.759765625, 0.159912109375, -0.30615234375, -0.474853515625, -0.1973876953125, -0.310546875, -0.1573486328125, -0.447998046875, -0.52587890625, 0.41455078125, 0.352294921875, -0.77001953125, 0.52197265625, 0.56591796875, 0.457275390625, 0.12115478515625, -0.0997314453125, 2.12109375, 0.346923828125, -0.11846923828125, -0.75830078125, 0.814453125, 0.81640625, 0.5380859375, -0.2103271484375, 1.689453125, 0.2020263671875, -0.6806640625, -0.509765625, -0.27197265625, 0.067626953125, -0.751953125, 0.611328125, 0.73828125, -1.349609375, 0.73193359375, 0.4404296875, -0.360107421875, 1.681640625, 0.403076171875, -0.6533203125, 0.671875, 0.442626953125, 1.921875, 1.1513671875, 0.642578125, 0.258056640625, 0.354248046875, -0.133056640625, -1.6923828125, -0.160400390625, -0.62255859375, 1.4951171875, 1.2509765625, -1.84765625, 1.8369140625, 0.728515625, 0.77587890625, -0.431884765625, -0.338623046875, -0.015655517578125, 1.0166015625, -1.52734375, -1.1787109375, 0.1629638671875, 0.6162109375, 0.346435546875, -1.5654296875, 1.8291015625, 1.142578125, 1.2470703125, 0.25927734375, 0.381591796875, 1.16796875, -0.331298828125, -0.82470703125, 1.099609375, 1.3330078125, 0.2666015625, 0.1575927734375, 0.67138671875, -0.1712646484375, 0.00567626953125, -0.27734375, 1.2255859375, -0.72900390625, 0.442138671875, -2.994140625, -0.70947265625, -1.4287109375, 0.57861328125, 0.1461181640625, 1.5009765625, -0.83203125, 0.6005859375, -0.429931640625, -0.94384765625, -1.5625, -0.5419921875, 0.1900634765625, 0.114501953125, 0.619140625, -0.99365234375, 0.85009765625, -0.34716796875, -0.082763671875, 0.044769287109375, -0.53466796875, 0.22216796875, 0.5341796875, 0.537109375, -1.0302734375, -0.84814453125, -0.98974609375, 1.7646484375, -0.00472259521484375, 0.91455078125, 0.6357421875, 0.2447509765625, -0.82470703125, -1.5517578125, 1.6533203125, -0.50341796875, -1.26171875, 0.7958984375, -1.302734375, 0.92236328125, -1.0244140625, -1.0146484375, -0.5869140625, -0.97265625, 0.6826171875, -0.94873046875, -1.4306640625, -1.2646484375, 1.4228515625, 0.3330078125, 0.378662109375, 1.6181640625, -2.029296875, -0.1630859375, -0.58447265625, 0.94140625, 0.1568603515625, -0.466796875, 1.458984375, 0.08636474609375, 1.4345703125, 0.353759765625, -0.54931640625, 0.720703125, 0.2432861328125, -0.00574493408203125, -0.423095703125, -0.0228118896484375, -0.447021484375, -0.046905517578125, 0.3671875, 0.25927734375, 1.3759765625, 1.1943359375, -0.189208984375, 1.3505859375, -0.1927490234375, -0.1365966796875, -0.6875, -0.849609375, -0.90234375, -0.1844482421875, -1.013671875, -0.76220703125, -0.452880859375, -0.0222930908203125, -0.47265625, -0.09954833984375, 0.2568359375, -0.1470947265625, 0.7744140625, 0.31298828125, 0.17919921875, -0.712890625, -0.432861328125, 1.1474609375, -0.438720703125, 0.73193359375, 1.5693359375, 0.94677734375, -1.48046875, 0.67236328125, 0.63671875, 0.388916015625, -1.103515625, -0.363037109375, -1.029296875, -0.4658203125, 1.2685546875, -1.1533203125, 0.271728515625, -1.4833984375, -0.76220703125, 0.09857177734375, -1.2939453125, 1.189453125, 0.1474609375, 1.091796875, 1.1298828125, 2.71484375, 2.392578125, -0.1688232421875, 0.66748046875, 0.0228424072265625, -1.2353515625, 0.07470703125, -1.7744140625, -0.63916015625, -0.9404296875, 1.57421875, -0.85302734375, -0.74462890625, -0.53466796875, -0.0231781005859375, 1.419921875, 0.342529296875, 0.0220794677734375, -0.80029296875, -0.69140625, 0.88916015625, -0.2357177734375, 0.28125, -0.370361328125, -0.84619140625, -1.2373046875, 0.16650390625, 0.261962890625, -2.16015625, 0.162109375, -0.78662109375, 0.7236328125, 0.87255859375, -0.626953125, -0.18896484375, 0.1478271484375, 0.40234375, 1.1318359375, -0.21923828125, -0.50390625, 0.132568359375, 0.1671142578125, -0.5283203125, -1.419921875, -0.35107421875, 0.572265625, -1.5107421875, -0.60009765625, 0.27490234375, -0.0633544921875, 0.6201171875, -0.026763916015625, 0.064453125, 0.88232421875, -0.79052734375, 0.37353515625, 0.329833984375, -0.0311126708984375, 0.92529296875, 0.2064208984375, -1.0712890625, 0.2587890625, -2.375, 0.05841064453125, 0.485595703125, 0.77783203125, -0.185302734375, 0.98828125, 1.4296875, 1.818359375, -0.5947265625, 0.299072265625, -0.31884765625, 0.5048828125, 0.1585693359375, -0.391845703125, 0.36962890625, -1.154296875, 1.1845703125, 0.055572509765625, -0.1373291015625, -0.970703125, 1.6943359375, 0.0799560546875, 0.0416259765625, 1.0556640625, 0.2264404296875, -0.5205078125, 2.421875, -0.4345703125, 0.06927490234375, -1.8994140625, -1.841796875, -0.455078125, 1.6123046875, 0.91357421875, -0.03472900390625, -1.0556640625, 1.634765625, 0.92138671875, -2.3125, -1.3251953125, -1.33203125, -0.2340087890625, -1.5546875, 0.150390625, 1.62109375, 1.546875, 1.328125, -1.013671875, -1.2978515625, 0.841796875, 0.376953125, -0.00865936279296875, 0.41015625, -0.87353515625, 0.568359375, 0.1683349609375, 0.92578125, -0.058258056640625, 0.1905517578125, 0.287841796875, -0.15283203125, -0.56884765625, -0.6708984375, 0.88916015625, 0.63525390625, 0.2841796875, 1.1796875, -0.492919921875, -0.469970703125, -0.482177734375, -0.10498046875, -1.0224609375, -0.5087890625, 0.1103515625, -0.271484375, -1.564453125, -2.04296875, 0.8681640625, 0.12841796875, 0.0255126953125, 0.75537109375, 0.2330322265625, -0.5146484375, -0.1785888671875, 0.341796875, -0.73486328125, -1.6201171875, -1.0517578125, -0.2186279296875, -1.0283203125, -0.410888671875, 0.16015625, -0.0728759765625, -0.59326171875, -1.6865234375, 0.892578125, 1.8017578125, -0.1937255859375, -0.9677734375, 1.4921875, 0.45947265625, -0.79248046875, 0.037384033203125, -0.3759765625, 0.818359375, -0.2607421875, -1.4345703125, -0.0693359375, -0.2578125, -0.6259765625, 0.29248046875, -2.041015625, -0.8525390625, 0.09979248046875, 0.0019254684448242188, 0.76953125, -0.457275390625, 0.763671875, 0.03851318359375, -0.525390625, -0.254150390625, 0.4453125, -1.2119140625, -0.438720703125, -0.255859375, -0.37548828125, -0.1986083984375, 0.44482421875, -0.75341796875, 0.83154296875, 0.0689697265625, 0.437255859375, 0.87646484375, 1.2490234375, 1.669921875, -0.300048828125, 0.09393310546875, -1.349609375, 0.63232421875, -2.294921875, -0.44482421875, -0.19970703125, 0.2193603515625, 0.8525390625, -0.358642578125, 1.908203125, -0.5126953125, 0.71435546875, -0.132080078125, -1.625, -0.86572265625, -1.375, -1.392578125, 0.403076171875, -1.404296875, -0.46875, 0.17724609375, -0.98974609375, 0.3203125, -0.332763671875, -0.97900390625, -1.0458984375, -0.016754150390625, 0.783203125, -1.45703125, 0.350830078125, 2.484375, 1.2890625, 1.5498046875, 1.185546875, -0.62255859375, 0.0477294921875, -0.98388671875, -0.468994140625, -0.150634765625, -0.60595703125, 0.88525390625, 1.1142578125, -0.412109375, -0.6982421875, -0.1358642578125, -1.837890625, -0.0283203125, -0.349609375, -0.31982421875, -1.1533203125, 0.64208984375, -1.390625, -1.3955078125, -1.9951171875, -1.0986328125, -0.859375, 1.966796875, 0.56640625, 0.6787109375, -0.40869140625, -1.123046875, 0.04803466796875, -1.7060546875, -2.06640625, 1.0009765625, 0.9453125, -0.370361328125, 1.69140625, 0.61181640625, -0.347900390625, -0.744140625, 1.732421875, -0.50341796875, -0.68359375, 0.84033203125, -0.59716796875, -0.53662109375, 1.5361328125, -0.548828125, 1.091796875, -1.2734375, 1.025390625, -0.17041015625, 0.463623046875, -0.62109375, -0.623046875, 0.896484375, 0.1466064453125, 0.76806640625, 1.517578125, -1.3701171875, -0.1083984375, -0.734375, 1.7529296875, -1.5859375, 0.293212890625, -0.59033203125, 0.77734375, 0.7392578125, -0.77392578125, -1.1240234375, 0.79296875, -0.2890625, 0.486083984375, -1.443359375, 1.2099609375, 0.337158203125, -1.318359375, -1.2841796875, -0.307373046875, 0.779296875, -0.736328125, -0.59814453125, 0.438232421875, 0.21533203125, 1.068359375, -0.389892578125, 0.7705078125, 0.1854248046875, 0.38232421875, -0.7080078125, 0.91015625, 0.491455078125, -0.5390625, -1.37109375, 0.2449951171875, -0.05487060546875, 0.186279296875, 0.326171875, -0.72509765625, -1.916015625, 0.325927734375, 0.86474609375, 1.65625, -0.94921875, -0.58984375, -1.1806640625, -0.58642578125, -0.66162109375, 0.3173828125, -0.6552734375, -0.03411865234375, -1.2919921875, 0.2685546875, -1.4375, 0.251953125, -0.9599609375, -1.2568359375, 0.92578125, -0.33740234375, 0.6279296875, 1.1484375, 0.056365966796875, -1.125, -0.6015625, 0.35693359375, -0.31689453125, -0.044525146484375, -0.08563232421875, 0.86474609375, -1.02734375, 0.499267578125, -0.5400390625, 0.018463134765625, -0.054229736328125, -0.032470703125, -1.2431640625, -0.3076171875, 1.376953125, -0.6767578125, 0.83740234375, -0.38232421875, -1.4072265625, -2.18359375, -0.6240234375, -1.1435546875, 0.263916015625, 1.2646484375, -1.5556640625, 1.5908203125, -1.298828125, -1.275390625, 0.98583984375, 1.1728515625, 0.70556640625, -0.10009765625, 0.82275390625, 0.98779296875, -0.034912109375, 1.447265625, 0.75146484375, 0.49853515625, 0.394775390625, -0.0533447265625, -0.63671875, -0.417724609375, 0.517578125, 1.3818359375, -0.416748046875, -0.303466796875, -0.379150390625, 1.8466796875, -0.72998046875, -1.486328125, -0.1185302734375, 3.005859375, -1.2509765625, 1.404296875, 0.990234375, -0.5849609375, -1.3466796875, 0.4873046875, -0.05126953125, 0.95556640625, 0.71533203125, 1.5341796875, 2.466796875, 0.1749267578125, -0.87060546875, -0.94482421875, 0.4521484375, 0.40576171875, 1.1923828125, -1.6572265625, -0.027130126953125, 1.6689453125, -0.021484375, -0.9443359375, -2.037109375, 1.01171875, -1.35546875, 0.33935546875, -2.4296875, 1.583984375, -0.1461181640625, 0.42626953125, 1.166015625, 0.98388671875, 0.08416748046875, -0.62158203125, -0.69384765625, -0.47998046875, -0.84912109375, 0.54150390625, -0.389404296875, -0.93310546875, -0.10504150390625, -0.46044921875, -0.8740234375, 0.1517333984375, -1.2255859375, 1.17578125, 0.22265625, -0.66943359375, -1.677734375, 1.703125, -0.09271240234375, -0.267578125, -1.974609375, 1.19921875, -0.55224609375, 1.9306640625, -1.0419921875, -0.031707763671875, 0.705078125, -0.96728515625, 0.625, 1.9404296875, 0.129150390625, 0.03778076171875, -0.85498046875, 0.90478515625, -0.322998046875, 0.19287109375, -0.10443115234375, 1.0322265625, -1.427734375, -0.431884765625, 0.463134765625, -0.1611328125, 0.5615234375, -0.1248779296875, 0.71044921875, -1.658203125, -0.6708984375, -0.42041015625, -0.69921875, -0.049102783203125, 0.54248046875, -0.68505859375, -0.54150390625, -0.055694580078125, -1.078125, 0.27880859375, 1.3369140625, 0.342041015625, 0.98291015625, -0.81689453125, -0.1719970703125, -1.744140625, -1.0302734375, 0.210205078125, 1.193359375, 0.067626953125, -1.48046875, 1.2646484375, 0.52392578125, -0.3720703125, 0.583984375, 1.8623046875, -1.4736328125, -0.75439453125, -0.27880859375, 0.5556640625, -0.2308349609375, 0.09112548828125, 1.142578125, 1.3173828125, -0.305908203125, 0.28466796875, 0.611328125, 0.133056640625, -0.11065673828125, 0.69482421875, -0.68603515625, -0.03607177734375, -0.81689453125, -0.69384765625, 0.541015625, -0.6083984375, -1.6181640625, 1.0439453125, -0.34912109375, -1.69140625, 0.2388916015625, 0.1077880859375, 1.5166015625, 0.055023193359375, 0.7841796875, 0.67138671875, -0.253662109375, 1.66015625, -0.55810546875, 0.10284423828125, 0.244140625, -0.873046875, -1.3671875, -0.1793212890625, -0.80615234375, -0.05865478515625, 1.576171875, -0.19189453125, 0.276123046875, 0.60791015625, -0.46435546875, -0.4443359375, -0.9375, -1.5751953125, -2.361328125, 0.2039794921875, 0.0267486572265625, 0.222900390625, -1.541015625, 1.09765625, 1.798828125, -1.072265625, -0.63427734375, 2.083984375, 0.37548828125, 0.2000732421875, -1.06640625, 0.31982421875, 0.546875, 1.9794921875, 0.2109375, 1.142578125, 0.2283935546875, 0.6376953125, -0.98388671875, 1.3828125, -0.70654296875, 1.5625, 2.58984375, 0.482177734375, 0.06591796875, 0.254638671875, -0.33740234375, 0.457275390625, 0.2489013671875, 0.46142578125, -0.51611328125, 0.1885986328125, 0.78759765625, -0.425048828125, 1.763671875, 1.2001953125, 0.360595703125, 0.41552734375, -1.6103515625, 0.0281219482421875, -0.72412109375, -0.1796875, -0.53564453125, -0.3310546875, -0.22607421875, -0.0706787109375, -0.63916015625, -0.3505859375, -1.0146484375, 0.42919921875, 0.447509765625, 0.1171875, -0.7158203125, 1.5693359375, -0.2264404296875, -0.1434326171875, 1.7216796875, -2.001953125, 1.2900390625, -0.31982421875, -0.00743865966796875, -0.96142578125, 1.193359375, -0.0204010009765625, -0.806640625, -0.82861328125, 1.494140625, -1.2578125, -1.7783203125, -0.9033203125, 1.04296875, 1.486328125, -0.1068115234375, 1.2724609375, -1.09765625, 0.98291015625, -0.10894775390625, -1.82421875, 0.39111328125, -1.12890625, 0.38525390625, 0.00754547119140625, 0.1512451171875, 0.8466796875, 0.251953125, -0.734375, -0.529296875, 0.63232421875, -0.57666015625, 1.1435546875, 0.07159423828125, -1.2275390625, -2.427734375, 1.8017578125, -2.021484375, 1.220703125, 1.310546875, 0.048126220703125, -0.9912109375, 0.208251953125, 1.2255859375, -0.144775390625, 0.2239990234375, 1.3173828125, -1.109375, -0.3134765625, -0.9208984375, 0.75, 2.28515625, 0.5, 1.525390625, -0.96240234375, -2.021484375, -0.2666015625, 0.27587890625, -0.64453125, -0.437744140625, 2.775390625, 1.0029296875, 0.4384765625, -1.921875, 0.57470703125, -1.10546875, 0.10198974609375, -1.0947265625, -0.98388671875, 3.431640625, 0.53564453125, 0.332275390625, -0.408447265625, -0.77783203125, -0.78662109375, -0.80810546875, -0.97998046875, -0.66064453125, 0.77978515625, 0.81591796875, 0.89501953125, 0.0111541748046875, 1.4150390625, -0.31005859375, -1.6162109375, 0.59033203125, -0.266845703125, 0.66748046875, 0.327880859375, -0.10931396484375, 0.78515625, 1.4619140625, 0.90576171875, -1.3994140625, 0.2178955078125, -2.462890625, -1.240234375, -0.187255859375, -1.78515625, -0.73388671875, 0.84619140625, -0.44091796875, 0.20361328125, -1.484375, -2.349609375, -0.09429931640625, -1.021484375, -0.77783203125, 0.3876953125, 0.62548828125, -0.6416015625, 0.384033203125, 0.0259552001953125, 0.0285491943359375, -0.8935546875, -0.61181640625, -0.2176513671875, 0.0316162109375, -2.2109375, -0.6865234375, 1.2236328125, -0.263427734375, -1.748046875, -0.387939453125, -1.03125, 0.05987548828125, -0.325927734375, -0.016845703125, 0.689453125, -1.9755859375, -0.2548828125, -0.6689453125, 1.6318359375, -0.69189453125, -1.2041015625, -0.45947265625, -0.5400390625, -0.236083984375, -1.1416015625, 0.90771484375, 0.5888671875, 0.037353515625, -1.1796875, 0.492919921875, 0.465087890625, 0.39111328125, 0.806640625, 0.1678466796875, -0.3818359375, 2.181640625, 1.01171875, 0.85986328125, 1.041015625, 1.0439453125, -0.4189453125, -0.56689453125, 0.039825439453125, 0.4189453125, 0.4150390625, 0.31298828125, 0.00969696044921875, -0.740234375, 0.87060546875, -0.333740234375, 0.1395263671875, 0.9775390625, 0.484619140625, 1.4228515625, -1.1591796875, -1.33984375, -0.2283935546875, 1.2890625, 1.30078125, 1.5390625, -1.3388671875, -0.184326171875, 0.6240234375, 0.94580078125, -1.3369140625, -0.031829833984375, 1.5322265625, 0.97412109375, 0.57177734375, -0.51123046875, 0.31884765625, -2.01171875, -0.416748046875, 1.0009765625, 0.285400390625, 0.9384765625, 0.2291259765625, 0.037872314453125, -0.9326171875, 0.51904296875, -1.2626953125, 0.529296875, -0.171875, 0.15771484375, -0.68701171875, -0.197021484375, 1.8974609375, -0.763671875, -0.21044921875, 0.5673828125, -0.007633209228515625, 0.250732421875, 1.453125, 0.7587890625, -0.7431640625, -0.87158203125, -0.908203125, -0.328857421875, -1.509765625, -1.4609375, 0.0301666259765625, 0.39990234375, -0.305908203125, 0.218505859375, 0.127685546875, 0.658203125, -0.45947265625, -0.57177734375, 0.1456298828125, -0.533203125, -1.7314453125, -0.393798828125, -1.1220703125, -0.09271240234375, 0.1727294921875, 0.56396484375, 0.72900390625, 2.216796875, 0.0465087890625, -0.34912109375, 0.6904296875, 0.8837890625, 1.013671875, 2.103515625, -0.444091796875, -1.3466796875, 0.2890625, -0.170654296875, 0.07415771484375, -0.87939453125, 0.5693359375, -0.049591064453125, -0.4560546875, 1.1611328125, 0.04156494140625, 1.0703125, -0.72412109375, -1.919921875, -0.728515625, -0.6552734375, -0.419189453125, -0.41796875, 0.6845703125, -1.6708984375, -0.119140625, -0.484619140625, -1.7548828125, -0.1231689453125, -0.85498046875, -0.5380859375, -1.1787109375, -0.6005859375, 0.311279296875, 0.59814453125, -1.7763671875, 0.03955078125, -0.1490478515625, -0.1552734375, -0.8056640625, -0.49853515625, 2.234375, 0.192626953125, -0.7587890625, -0.0025768280029296875, 0.64892578125, 0.480712890625, -0.55712890625, -0.362548828125, -1.1455078125, 0.6455078125, 1.1416015625, 1.462890625, 1.83984375, -1.3046875, -1.2822265625, -0.450439453125, 0.42724609375, 0.630859375, -0.5341796875, 0.2044677734375, 0.2454833984375, -0.8515625, 0.59375, 2.10546875, 1.16015625, -1.2626953125, -0.6552734375, 1.130859375, 2.232421875, 1.17578125, 0.09136962890625, 0.60595703125, -0.7568359375, -0.6025390625, 0.058319091796875, -1.1689453125, 0.1397705078125, 0.36181640625, -0.043975830078125, 2.12109375, 0.6630859375, -0.2344970703125, 1.064453125, -0.642578125, 1.0625, -2.521484375, -1.72265625, 0.2802734375, -0.055206298828125, -0.7880859375, -0.96875, 1.158203125, 0.64599609375, 0.5078125, -1.6201171875, -0.12225341796875, -0.328369140625, 0.279541015625, -0.67724609375, -0.009765625, -1.6025390625, 0.62353515625, -1.4873046875, 1.802734375, 0.4892578125, -2.021484375, -1.2978515625, -0.6982421875, 0.1435546875, -0.4453125, -0.73828125, -0.1412353515625, -1.16015625, 1.0205078125, -0.61181640625, -0.345947265625, 1.0244140625, 0.2496337890625, 1.5859375, -0.4677734375, 1.0029296875, -0.7529296875, -0.587890625, 0.21630859375, -0.263671875, -0.51708984375, 1.40234375, 0.71240234375, -1.052734375, 0.491943359375, -0.50439453125, -0.84130859375, 1.267578125, 1.720703125, -0.943359375, 0.96435546875, -0.2291259765625, -0.443115234375, -1.80078125, 0.615234375, 1.369140625, 0.71728515625, 1.228515625, -1.1015625, -0.75, -0.447998046875, -2.025390625, -1.24609375, -0.9775390625, 0.4052734375, 0.068603515625, -0.2078857421875, 0.88037109375, 1.2646484375, -0.8740234375, -0.125732421875, 1.5087890625, -0.66259765625, -0.5869140625, 0.35791015625, 0.485107421875, 1.62890625, 0.3701171875, -0.19580078125, 1.1025390625, -0.49560546875, -0.908203125, 0.89501953125, 0.06732177734375, 0.845703125, 0.50537109375, -0.28369140625, -2.021484375, -0.546875, 0.1300048828125, 0.490966796875, -2.41796875, 0.59423828125, -1.1708984375, 0.49755859375, -0.10504150390625, -1.0869140625, -0.7294921875, 2.6953125, -0.96484375, 1.6318359375, -0.5498046875, 0.229736328125, 1.12890625, -0.362548828125, -1.037109375, -1.3359375, -1.1083984375, 0.998046875, 0.85107421875, 0.779296875, -1.873046875, 0.21337890625, 0.35009765625, 0.8232421875, 0.370361328125, 0.96142578125, 0.8671875, -0.28564453125, 0.89306640625, 0.98095703125, -0.2044677734375, 0.15234375, 0.90234375, 0.53173828125, 1.11328125, 0.50146484375, 0.19091796875, -1.392578125, 2.271484375, -0.1285400390625, -0.412109375, -0.58984375, -0.074951171875, 0.8896484375, -0.283447265625, 0.62353515625, -0.96533203125, 0.423095703125, -0.10064697265625, 0.344482421875, 0.908203125, 0.875, 0.348876953125, -0.357421875, 0.236328125, -0.4482421875, -0.1839599609375, -0.036956787109375, -0.07647705078125, 0.02557373046875, -0.371337890625, 0.89697265625, -3.08203125, 0.57958984375, 1.521484375, -0.39306640625, -0.060089111328125, -1.5400390625, -0.767578125, -0.08447265625, 0.06585693359375, -0.50927734375, 0.479248046875, 0.00435638427734375, 0.74365234375, 1.75390625, 0.391357421875, 0.32568359375, -1.2470703125, 0.62109375, 0.6259765625, -0.59716796875, -0.176025390625, -0.248046875, -0.30810546875, -0.188232421875, 0.277587890625, 1.1123046875, -0.21875, -0.1424560546875, 0.68359375, 1.4306640625, -0.57373046875, -0.132080078125, 0.94921875, 0.1790771484375, -1.0830078125, 1.6826171875, 0.5595703125, 1.2138671875, 1.2001953125, 1.0126953125, 0.16552734375, -0.71630859375, -0.11407470703125, -2.375, 0.77294921875, 1.654296875, 0.200439453125, 0.9150390625, -0.319580078125, -0.183349609375, 0.129638671875, 1.619140625, 1.5048828125, -1.5078125, 0.0753173828125, -0.79443359375, -0.9951171875, -1.4853515625, -1.5556640625, 1.8369140625, -0.93212890625, -0.95458984375, -0.82373046875, -0.66748046875, 0.1800537109375, -0.181884765625, 0.41845703125, -1.18359375, 2.12109375, 0.037139892578125, 0.4404296875, 0.385009765625, -0.287109375, 1.2216796875, 0.385009765625, -1.466796875, 1.044921875, 0.42333984375, 2.01171875, -0.2283935546875, 0.435302734375, 1.166015625, -0.82763671875, -1.30078125, 2.181640625, 0.039825439453125, -0.890625, -1.5390625, 0.0011844635009765625, -0.89306640625, 1.337890625, -0.9140625, 0.63134765625, 0.82421875, 0.136962890625, -0.73291015625, -0.60888671875, 0.3916015625, -1.0078125, 0.70458984375, -1.4599609375, 1.9814453125, -1.91015625, -0.357421875, -0.81884765625, 2.7421875, 0.006610870361328125, 0.365234375, -0.7578125, -0.0924072265625, -1.40234375, -0.5, 0.04010009765625, -0.1302490234375, -0.0654296875, -0.1943359375, 0.81689453125, 0.19921875, -0.07122802734375, -1.2724609375, 1.12890625, 0.81787109375, 0.059112548828125, 0.25634765625, 0.1480712890625, -0.49169921875, -0.64892578125, 0.69921875, 0.0478515625, -1.21484375, 1.2275390625, 0.302734375, 1.4052734375, -0.76318359375, 0.6904296875, 0.12420654296875, 0.4814453125, -0.76806640625, -0.050994873046875, 1.0263671875, -2.09765625, 0.7998046875, -0.1708984375, -1.541015625, -0.685546875, -0.27294921875, -1.1591796875, -2.740234375, -0.1729736328125, -0.382568359375, 0.447021484375, -1.0712890625, 0.05267333984375, -1.3798828125, 1.310546875, -0.25146484375, 0.1083984375, 1.3583984375, -0.6298828125, 1.73046875, -0.476318359375, 0.349609375, -1.111328125, -0.366455078125, -1.490234375, -0.06805419921875, -0.419189453125, -1.9716796875, 0.603515625, -0.481689453125, 0.2802734375, 0.4052734375, 0.400390625, -0.169677734375, -0.62060546875, -1.396484375, 0.8037109375, 0.2447509765625, 0.09552001953125, 0.47607421875, -1.5615234375, -0.199462890625, 0.7431640625, 0.9697265625, 1.6181640625, 0.96826171875, -0.1544189453125, -0.88232421875, -1.1259765625, -0.06036376953125, -1.2861328125, 0.369140625, 1.1455078125, 0.75537109375, -0.8837890625, 1.3076171875, 1.2421875, -0.181396484375, 2.041015625, 0.061065673828125, 0.57958984375, -1.73046875, -0.466064453125, -0.207275390625, -2.146484375, 0.377685546875, 1.3720703125, -0.0660400390625, 0.38525390625, -0.1300048828125, -0.260986328125, -0.264404296875, 1.46875, -0.254638671875, -0.5732421875, 0.07147216796875, 0.435302734375, 1.765625, 0.1810302734375, 0.9716796875, -0.5595703125, -0.61279296875, 0.279541015625, 0.54052734375, -1.6181640625, -0.114501953125, 1.91796875, 0.09423828125, 0.86083984375, 1.26953125, -1.923828125, 0.19775390625, 0.96240234375, 0.28662109375, -0.410400390625, 0.6669921875, -0.875, -1.419921875, 0.5830078125, 0.005859375, 0.01543426513671875, -0.9892578125, -0.67333984375, 0.74072265625, -0.229248046875, 0.040435791015625, -0.1876220703125, 0.3203125, 0.81689453125, -0.5478515625, -0.011505126953125, -2.533203125, -1.318359375, -1.142578125, 0.58447265625, -0.484375, -0.3876953125, -0.87353515625, 0.2310791015625, 2.03125, -0.603515625, 1.5244140625, 0.60986328125, 0.51171875, 0.873046875, -0.1590576171875, -0.2117919921875, -3.25, -0.8662109375, -1.5517578125, 1.783203125, 1.1591796875, 0.243408203125, -0.438720703125, 0.09039306640625, 1.0068359375, 1.0126953125, -1.185546875, -1.8876953125, -0.1336669921875, -1.3173828125, 2.53515625, -0.1605224609375, -0.771484375, 0.56640625, -1.431640625, 0.97509765625, -0.290771484375, -1.1328125, -0.486328125, -1.7822265625, 2.10546875, -1.1171875, 0.74072265625, 1.177734375, -0.60888671875, -1.4619140625, -0.52685546875, -0.004791259765625, 1.5341796875, -0.53076171875, 0.0634765625, -1.056640625, -1.4462890625, -0.98583984375, 0.291748046875, 2.06640625, 0.08343505859375, 0.394287109375, 0.398193359375, 0.65185546875, 0.319091796875, 1.3876953125, -1.056640625, 0.84814453125, -0.434814453125, -0.10595703125, -0.61474609375, -0.26513671875, -0.5576171875, 0.7421875, 0.58984375, 0.51953125, -0.8125, -0.81494140625, 0.65478515625, 0.26904296875, 0.86279296875, -1.1025390625, -0.8583984375, 0.0117034912109375, 0.6259765625, 0.1494140625, -1.2021484375, 1.220703125, -1.1240234375, -2.2265625, 0.135986328125, -1.6220703125, 0.3115234375, -0.2181396484375, -0.320556640625, 0.294189453125, 0.84326171875, 0.51708984375, 0.78173828125, 0.70751953125, -0.71875, -0.1534423828125, -0.55126953125, -1.0126953125, 1.2705078125, -0.68701171875, -0.80322265625, 1.619140625, 0.57177734375, 0.1890869140625, 1.1357421875, -0.6552734375, 0.050994873046875, 0.54052734375, 1.1015625, -0.487060546875, -0.349609375, 0.1358642578125, 0.36962890625, -0.865234375, 0.1636962890625, 0.48095703125, -1.5029296875, -0.257568359375, -0.393310546875, 0.419189453125, 1.083984375, 0.1427001953125, -2.4296875, 0.9306640625, 0.34716796875, -0.330322265625, 1.5859375, -2.015625, -1.5859375, -0.2298583984375, 0.279541015625, 1.5205078125, 0.349365234375, 2.205078125, 0.033477783203125, -0.19921875, -0.490966796875, 0.388916015625, 0.2314453125, -1.4775390625, -0.311279296875, 1.03125, -0.1424560546875, -0.6923828125, 0.256591796875, 0.77197265625, 1.505859375, -0.75830078125, -1.9248046875, -0.266845703125, 0.00164031982421875, 2.197265625, -0.59033203125, 0.1661376953125, 0.00531005859375, -0.849609375, -0.05462646484375, -0.369384765625, 1.140625, -0.480224609375, 0.51025390625, 1.2841796875, 1.6259765625, -1.8291015625, 0.66552734375, -0.05780029296875, -2.7578125, -0.9833984375, -0.93603515625, 0.34423828125, -0.366455078125, 0.64013671875, -0.32861328125, 0.5849609375, -0.50390625, 0.282958984375, -2.171875, 0.97802734375, -2.458984375, 0.385009765625, 0.8369140625, -0.619140625, -1.3876953125, 0.158203125, -0.008758544921875, 0.802734375, 0.1466064453125, 0.344970703125, -1.517578125, 0.1605224609375, 0.414794921875, 1.486328125, 1.46875, 0.1859130859375, 0.818359375, 1.3544921875, 0.51953125, -1.8408203125, -0.5810546875, 0.783203125, -1.2470703125, 1.08984375, -1.173828125, -0.236328125, -1.6494140625, -1.1259765625, -0.34326171875, 2.298828125, 0.763671875, -0.54248046875, 0.63623046875, -0.76513671875, -0.62353515625, -1.03125, 0.395263671875, -0.007320404052734375, 1.1240234375, 0.8125, 0.580078125, 0.7685546875, 1.2373046875, -0.3330078125, -1.744140625, 1.712890625, 1.3046875, -0.68505859375, -1.2119140625, 0.95947265625, -0.716796875, -0.2135009765625, -0.9794921875, 2.861328125, -0.29248046875, -0.262451171875, -0.99560546875, -0.43212890625, -0.67919921875, -0.7958984375, -0.0975341796875, -1.759765625, -0.716796875, -0.385009765625, 0.6572265625, -0.6953125, 2.1875, 0.0416259765625, -2.015625, -0.1715087890625, -2.259765625, -0.7373046875, 0.7021484375, 0.7978515625, 1.962890625, -1.28125, 1.3251953125, -0.17529296875, -0.859375, -0.08782958984375, 0.5205078125, 0.262451171875, 1.3837890625, 1.03515625, -1.6533203125, -0.88427734375, -1.1044921875, -0.057464599609375, -0.79296875, 1.0537109375, 0.366455078125, 0.55029296875, 1.91796875, 0.72265625, 0.07171630859375, 0.277099609375, -0.7021484375, -0.53662109375, 0.0277252197265625, -0.65576171875, 0.7919921875, -0.467041015625, -0.708984375, -0.65478515625, 0.0010118484497070312, 1.3388671875, 0.339111328125, -1.033203125, 1.1474609375, -1.30078125, 1.0791015625, 1.2763671875, -2.154296875, 0.1427001953125, 1.0498046875, 0.2342529296875, -1.54296875, 1.4375, 0.07598876953125, -1.5439453125, -0.99853515625, 1.0810546875, -0.38720703125, 1.318359375, -0.14013671875, 0.04443359375, 0.15869140625, 0.76416015625, -0.353271484375, -1.060546875, 0.51123046875, 0.27294921875, -0.986328125, 0.654296875, 0.73583984375, -0.1334228515625, 0.310791015625, -0.65625, -0.2073974609375, 0.59716796875, -2.107421875, 0.269287109375, -1.0087890625, -0.6630859375, 0.5634765625, 0.0008792877197265625, 0.68408203125, 0.310302734375, -0.482177734375, -0.359130859375, -0.619140625, 1.1376953125, 0.63427734375, -0.86669921875, -0.408203125, 0.74755859375, -1.345703125, -0.666015625, 0.73486328125, -0.64111328125, -2.13671875, -0.292236328125, -0.22119140625, -0.00514984130859375, 1.1953125, 1.94921875, -0.441650390625, -1.2119140625, 0.9619140625, 0.2445068359375, -0.1317138671875, -0.350830078125, -0.802734375, -1.2412109375, -1.57421875, -0.87744140625, 0.1412353515625, 0.69677734375, -1.2890625, 0.41552734375, 0.018585205078125, 1.58203125, 0.1943359375, -0.052734375, 0.2408447265625, -1.2763671875, -0.81982421875, -1.091796875, -0.54833984375, -0.0020999908447265625, -1.728515625, -0.1951904296875, 0.65673828125, 0.77587890625, 0.75732421875, -0.64453125, 1.919921875, 1.130859375, -0.444091796875, 0.90087890625, 1.7998046875, -0.58837890625, -0.8662109375, 1.7451171875, -1.060546875, -0.256103515625, -0.65234375, 0.0284423828125, 1.259765625, 0.73193359375, -0.74267578125, -0.873046875, 0.07177734375, -1.3408203125, 0.9326171875, -1.365234375, -0.6103515625, -0.74951171875, 0.402587890625, 1.1005859375, -1.5732421875, 0.94384765625, -0.5732421875, -0.947265625, 0.304931640625, 0.23681640625, -0.255126953125, -1.318359375, -1.341796875, -0.56103515625, -0.275634765625, -1.828125, 0.54833984375, 0.10186767578125, 1.708984375, -1.2236328125, -0.191650390625, -0.0703125, -0.1160888671875, 1.0615234375, 1.86328125, -1.015625, -0.0018053054809570312, -0.1505126953125, 1.99609375, 0.084228515625, 0.106201171875, 0.12109375, -3.576171875, -1.0888671875, -0.6201171875, -0.75, -2.1953125, -1.068359375, -0.5771484375, -0.173828125, -2.576171875, 1.5078125, 0.68359375, -0.78662109375, 0.8515625, 0.81396484375, 0.94970703125, 0.11248779296875, -0.783203125, -0.3486328125, 0.794921875, 1.2177734375, -1.0546875, 1.2080078125, 0.215087890625, -1.033203125, 0.60400390625, 0.62451171875, -1.9892578125, 1.0087890625, 0.76708984375, -0.365234375, -0.053741455078125, 0.42578125, 1.21484375, 1.1435546875, 0.2457275390625, 1.740234375, 0.34619140625, 0.505859375, 0.52490234375, 1.4443359375, 0.1953125, 0.8447265625, -0.2073974609375, -0.048980712890625, 1.1279296875, 1.5048828125, 1.046875, -0.2227783203125, 0.27734375, -1.41796875, -2.12109375, 0.52490234375, -0.05926513671875, -0.247802734375, -1.54296875, -0.1522216796875, -1.4248046875, 0.2362060546875, 0.34912109375, 1.8388671875, 1.173828125, -1.0078125, 1.0380859375, 0.1396484375, -0.416015625, 0.154296875, 0.304931640625, -1.1728515625, 1.04296875, 1.564453125, -0.2200927734375, 0.55126953125, 0.30322265625, 0.237060546875, 0.294921875, 0.5322265625, -0.447265625, 0.91650390625, -1.20703125, -0.77734375, -0.5419921875, -2.22265625, -0.01227569580078125, 1.947265625, -0.96484375, -1.1640625, 2.169921875, 0.8125, 0.9853515625, 0.73046875, -1.544921875, -1.0810546875, 1.7783203125, -1.052734375, -0.02423095703125, -0.186279296875, 0.34130859375, 0.164794921875, -0.32763671875, 0.92822265625, -1.0654296875, 0.485595703125, -0.71044921875, 1.23046875, -0.30615234375, 1.3193359375, -0.0601806640625, 1.1123046875, 0.81298828125, -0.5634765625, 1.216796875, 0.38916015625, 1.6123046875, 0.78076171875, -0.396484375, 0.7890625, -0.153076171875, -1.70703125, -0.1763916015625, 1.2236328125, 1.388671875, 0.379638671875, -0.6474609375, -1.587890625, -0.1685791015625, 0.9912109375, 0.417236328125, 0.69775390625, -1.14453125, 0.329833984375, 0.366455078125, 0.183837890625, -1.9794921875, -0.382080078125, 0.7109375, 0.00765228271484375, 0.2099609375, -0.2249755859375, 1.392578125, -0.64794921875, -0.1561279296875, -0.62939453125, -0.253662109375, 1.1240234375, -1.5869140625, 1.056640625, 0.52099609375, 0.1995849609375, 1.884765625, -0.229736328125, 0.3193359375, 0.54345703125, -0.91845703125, 0.11138916015625, -0.154052734375, 1.3544921875, -0.611328125, 0.2017822265625, -0.21435546875, -1.0439453125, 0.248779296875, 0.11126708984375, -0.50341796875, -0.137939453125, 1.6103515625, -0.2022705078125, 0.379150390625, 1.34765625, 1.3134765625, 1.126953125, 0.52734375, -0.54541015625, 1.625, -0.74755859375, -0.87548828125, 0.74169921875, 1.2119140625, 1.90234375, -0.191162109375, 1.4150390625, -0.439453125, -0.48046875, -0.481201171875, -1.4873046875, -0.5537109375, -1.4443359375, -1.0400390625, -0.0682373046875, -0.68701171875, 1.216796875, -1.005859375, 0.80712890625, 0.55419921875, 1.796875, 0.019073486328125, 1.119140625, 0.2376708984375, -1.3125, -1.0, 1.1689453125, -0.609375, -0.369384765625, 1.9365234375, 1.173828125, 1.28515625, -0.14892578125, 0.26416015625, 0.0706787109375, 1.4873046875, 0.5166015625, -0.231201171875, -0.256591796875, -0.8486328125, -0.65185546875, -1.322265625, -0.1708984375, 0.88916015625, 0.8115234375, -0.1824951171875, -1.0546875, -0.033966064453125, 0.08099365234375, 0.018280029296875, -1.3125, -1.19140625, -0.80078125, 0.2332763671875, -1.7890625, -2.166015625, -0.63916015625, -0.6376953125, 1.103515625, -0.80615234375, -0.54736328125, -0.484130859375, 0.83203125, 1.4609375, -0.93115234375, -1.1787109375, -2.228515625, -1.765625, -0.5576171875, -1.3515625, 0.247802734375, 1.6435546875, 0.01033782958984375, -0.47509765625, 0.164794921875, 0.251953125, -1.1044921875, 0.401123046875, -1.25, 0.0384521484375, -0.57861328125, 0.6240234375, -1.2861328125, -0.2357177734375, -2.015625, 0.09759521484375, -0.53173828125, 0.6435546875, -1.0576171875, -1.1767578125, -1.4384765625, -0.517578125, -1.0361328125, 0.857421875, 0.5107421875, -0.036712646484375, 0.64599609375, -1.5146484375, -0.69091796875, 0.37939453125, -0.6884765625, 3.138671875, 0.458251953125, -0.175048828125, 0.77734375, -1.8740234375, 0.1361083984375, -0.63232421875, -0.92138671875, 2.16015625, -0.68701171875, -0.65673828125, 1.4501953125, 1.3271484375, -1.205078125, -2.212890625, -0.373779296875, 0.63232421875, 0.51953125, -1.4775390625, 0.74658203125, 0.0689697265625, 0.8203125, -1.09375, 0.038360595703125, -1.111328125, -1.0146484375, 0.291259765625, 0.80126953125, -0.47119140625, -0.4130859375, -0.818359375, 0.434814453125, 0.052703857421875, -0.10723876953125, 0.163330078125, 0.301513671875, 0.157958984375, 0.318359375, 0.67822265625, -0.80810546875, 0.89599609375, 1.154296875, -0.34130859375, 3.501953125, -0.6259765625, 1.0869140625, 0.9970703125, -0.0140533447265625, -0.42529296875, 0.1280517578125, 0.18701171875, 0.63671875, -1.05078125, -2.451171875, -0.2158203125, -0.23291015625, -0.53076171875, 0.262451171875, -0.9052734375, -0.2012939453125, 1.169921875, 0.02996826171875, -0.357177734375, 0.163330078125, 0.483642578125, -0.92138671875, 0.293212890625, 1.3251953125, -0.213134765625, 1.1591796875, 0.91162109375, 0.0738525390625, 0.21533203125, 0.69970703125, -0.2159423828125, -1.02734375, 0.07672119140625, 0.2325439453125, 1.373046875, 0.313232421875, 1.3818359375, 0.9189453125, -1.9873046875, 0.17578125, 0.5361328125, 0.88427734375, -0.045623779296875, -2.267578125, 0.9453125, 0.08868408203125, 1.0986328125, 0.09124755859375, 0.662109375, -0.05078125, 0.1617431640625, -0.36962890625, 0.755859375, -1.40234375, 1.728515625, 0.8203125, 0.82861328125, 0.372802734375, -1.4013671875, 1.4765625, -0.178466796875, -0.2783203125, -0.880859375, 0.41845703125, 0.11761474609375, -1.3994140625, 0.2254638671875, -0.2093505859375, 1.150390625, -0.7265625, 0.426513671875, -0.55615234375, -2.587890625, 0.28369140625, -0.499267578125, -0.58203125, 0.408447265625, -0.1265869140625, -1.68359375, -1.6103515625, -0.65283203125, -0.7353515625, 0.09063720703125, -0.93017578125, 2.412109375, -1.0087890625, 1.2744140625, 1.0107421875, -0.85302734375, 0.440185546875, -0.884765625, -0.0017337799072265625, 0.7724609375, 1.7587890625, 2.240234375, 0.055389404296875, 0.8271484375, -0.350830078125, -1.107421875, 0.615234375, 0.51806640625, -1.1044921875, -0.32763671875, -0.76806640625, -0.1309814453125, -0.6953125, 3.15234375, -0.1436767578125, 0.061004638671875, -1.720703125, 2.791015625, 0.6787109375, -2.08203125, 0.223876953125, -1.4375, 1.15625, 0.86376953125, -1.287109375, -1.275390625, 1.4619140625, -0.86572265625, 2.017578125, -0.0208282470703125, -0.403076171875, -0.11737060546875, -0.67431640625, -0.208984375, -1.873046875, -0.6064453125, 1.2373046875, -0.537109375, -1.046875, 0.5166015625, -0.75390625, -1.33984375, 0.99609375, 0.041229248046875, -0.07904052734375, 0.3818359375, 1.08203125, -0.159912109375, -0.31982421875, -0.2276611328125, 0.53369140625, 0.89697265625, 0.86328125, -0.3388671875, 0.0875244140625, 0.424072265625, -0.12127685546875, 0.301513671875, -0.481689453125, 0.5712890625, -0.81494140625, -0.55712890625, -0.10894775390625, -1.8125, 0.640625, 0.806640625, 0.1954345703125, -0.260498046875, 2.572265625, -0.5712890625, -0.137451171875, -0.9990234375, -0.392333984375, -1.02734375, -1.06640625, 0.465087890625, -0.383056640625, 0.87109375, 0.1568603515625, -0.259033203125, 1.3447265625, -1.017578125, -0.12127685546875, 0.09088134765625, -0.28173828125, 0.65380859375, 0.55126953125, 1.2607421875, 0.83203125, -0.9169921875, -0.03778076171875, 0.6484375, 0.01445770263671875, 1.314453125, 2.15234375, -0.090087890625, 0.281982421875, -0.060546875, -1.4228515625, 0.8857421875, -0.62939453125, -0.81201171875, 0.5986328125, 0.208740234375, -0.7939453125, -0.51025390625, -1.326171875, -1.1162109375, -0.65576171875, -0.1766357421875, -0.2998046875, -0.759765625, 1.6552734375, 0.362060546875, -0.111328125, -1.7138671875, -0.49169921875, -0.05364990234375, -0.25244140625, 0.896484375, 1.201171875, -1.470703125, 0.16357421875, -0.00909423828125, 0.52001953125, -0.48095703125, 1.2607421875, -0.8193359375, -0.052093505859375, -1.5361328125, -1.1826171875, -0.319580078125, 0.1422119140625, -0.6591796875, 0.6953125, -0.2088623046875, -0.14013671875, -0.494384765625, 0.362060546875, 2.40625, 0.70654296875, 1.1640625, 2.587890625, 0.47509765625, -0.568359375, 3.361328125, 0.134521484375, -0.05902099609375, -0.0176849365234375, 0.5849609375, -0.86572265625, 0.09344482421875, -0.1959228515625, -0.939453125, -0.8740234375, 0.740234375, -3.125, 0.06805419921875, 0.62255859375, 1.0517578125, 0.62939453125, 0.98193359375, 0.96435546875, 0.474609375, -1.0224609375, -1.841796875, 0.2880859375, 1.09375, 0.053955078125, -1.373046875, -1.88671875, -1.3330078125, 0.143798828125, -1.0087890625, -0.59228515625, -2.16015625, 1.6640625, 1.708984375, 1.2568359375, -0.5078125, -0.927734375, -0.55712890625, 0.91064453125, 0.409423828125, 1.6044921875, -1.5810546875, -0.90283203125, 1.29296875, 1.8642578125, -1.296875, -1.4658203125, 1.2880859375, 0.78857421875, 1.236328125, -0.0225067138671875, 0.0175018310546875, -0.137939453125, 1.0185546875, -0.7763671875, -2.009765625, 1.2529296875, 2.056640625, 1.8125, -0.39501953125, -0.1646728515625, -0.1815185546875, -0.88037109375, 1.1142578125, 0.7734375, -0.97265625, -0.5029296875, 2.404296875, -2.009765625, 0.78515625, 0.022979736328125, 1.19921875, 0.904296875, 0.1776123046875, 0.032867431640625, 1.4384765625, 0.1334228515625, -0.35498046875, 1.228515625, 0.42333984375, -0.380126953125, 0.328369140625, 0.2255859375, 0.681640625, -0.1324462890625, 1.447265625, -1.072265625, -0.2364501953125, -1.6435546875, -1.216796875, 0.66552734375, -0.4951171875, 0.2132568359375, -0.2037353515625, 1.1201171875, -1.1572265625, -0.6767578125, -0.3212890625, -0.466796875, 0.55517578125, -1.1279296875, 2.0703125, -2.130859375, 0.63330078125, 1.0263671875, 0.327880859375, 1.9189453125, 1.19921875, 0.57373046875, -0.1964111328125, -1.4677734375, 0.420166015625, 1.57421875, -0.465576171875, -0.81201171875, 0.7236328125, -0.85595703125, 1.814453125, 0.6240234375, -0.67724609375, -0.300048828125, -0.72998046875, -0.288818359375, 0.354248046875, 0.408447265625, 1.2724609375, -1.6162109375, -0.08221435546875, 1.2060546875, -0.57861328125, -0.55419921875, 2.298828125, 0.1300048828125, 0.67919921875, -1.3583984375, -0.135009765625, 0.8896484375, -0.5478515625, 0.004425048828125, -0.78369140625, 0.09771728515625, -0.904296875, -0.68212890625, -0.42919921875, 1.529296875, 0.6552734375, 1.4267578125, 0.98583984375, 2.16015625, 2.1875, 0.286865234375, -1.2021484375, -0.364501953125, 0.96533203125, -0.2208251953125, 1.1181640625, -0.1937255859375, -1.3056640625, -0.25732421875, 0.01224517822265625, -3.3671875, 0.55517578125, 1.7021484375, -0.429443359375, -1.287109375, -1.0458984375, 0.61767578125, 0.1365966796875, 1.3623046875, -2.046875, 0.93310546875, 0.626953125, -0.75146484375, -0.2335205078125, -0.050079345703125, -0.1436767578125, 0.01715087890625, -1.0947265625, -1.4736328125, 1.1279296875, -1.275390625, 1.80078125, -0.53564453125, 0.63916015625, 0.235107421875, 0.56982421875, -1.91015625, 0.026031494140625, -0.383056640625, 1.595703125, -0.767578125, -0.17138671875, -0.40771484375, -2.4453125, -1.2373046875, 1.8251953125, 1.134765625, 0.0224151611328125, -0.1114501953125, -0.76708984375, -0.9375, -1.2880859375, 0.54248046875, -0.0838623046875, 0.191162109375, 0.79541015625, -1.9599609375, -0.244873046875, -0.007160186767578125, 0.9755859375, -0.1820068359375, 0.341796875, 0.1328125, 0.375732421875, 0.155517578125, -0.281494140625, -1.8994140625, -0.7197265625, -0.66796875, 0.264892578125, -0.061798095703125, -0.41748046875, 0.12066650390625, 1.21875, 0.5166015625, -0.061859130859375, -1.755859375, 1.5078125, -1.4833984375, 0.405029296875, -0.53857421875, -1.109375, 0.9833984375, 0.8515625, 0.58837890625, -1.0224609375, 0.88525390625, -1.5830078125, -3.26171875, -0.09039306640625, 0.359619140625, -0.7900390625, 0.91162109375, 1.4853515625, 0.5439453125, -0.6572265625, 0.2042236328125, -1.0224609375, 0.1993408203125, 0.7880859375, -1.154296875, -0.017578125, 1.4033203125, -0.15966796875, 1.1513671875, 0.08251953125, -2.193359375, -0.91748046875, 0.2445068359375, 0.5419921875, -0.49365234375, 0.41455078125, -0.11151123046875, 1.677734375, 0.028106689453125, -1.787109375, 0.6875, -0.87158203125, 0.51025390625, -0.51513671875, -1.8056640625, -1.5009765625, 1.5517578125, 0.29541015625, -0.126953125, -0.6025390625, -0.138427734375, 0.424072265625, -0.84814453125, -1.0859375, -1.546875, -1.4404296875, 1.46484375, -0.8857421875, 1.693359375, 0.19921875, -1.2548828125, -0.5029296875, -0.88037109375, 1.810546875, 0.6015625, 0.436767578125, -0.366455078125, 2.16796875, -0.75390625, -2.556640625, -0.261962890625, -0.11260986328125, 0.061492919921875, 1.5478515625, -0.70703125, -0.176025390625, 1.234375, 0.328857421875, 0.65966796875, -0.2142333984375, -0.3369140625, -0.37451171875, -0.89111328125, 1.3564453125, 0.802734375, -0.13623046875, -1.466796875, -0.6728515625, -0.236083984375, 0.051239013671875, -0.60302734375, 0.25146484375, 0.55078125, 0.9794921875, 0.0214691162109375, -0.96435546875, 1.126953125, -0.447998046875, 0.0728759765625, -1.677734375, -0.4296875, -1.017578125, -0.85791015625, 1.0244140625, 0.8779296875, 1.0380859375, 0.0188751220703125, -1.708984375, -1.46484375, -1.0947265625, 0.01090240478515625, -0.48291015625, -1.1201171875, 0.121337890625, 0.32373046875, -0.69287109375, 0.430908203125, -0.88037109375, -1.75390625, -2.2109375, -0.67431640625, 1.4541015625, -1.3203125, 1.4326171875, -0.06036376953125, 1.2119140625, 0.75244140625, -2.197265625, -2.044921875, 0.476806640625, -0.043426513671875, 0.24462890625, 1.1318359375, 0.8935546875, -0.0635986328125, -2.806640625, 0.7314453125, -1.1240234375, -0.1617431640625, 0.11114501953125, -1.0126953125, 0.61767578125, 0.03662109375, 0.7099609375, -2.02734375, -0.88623046875, -0.12158203125, -0.6123046875, 0.3828125, -1.279296875, 0.25341796875, -0.0950927734375, 0.486572265625, 0.62939453125, -0.90380859375, -0.3984375, 0.6806640625, -0.005931854248046875, -1.0029296875, -0.0313720703125, -0.2353515625, 0.1695556640625, -0.424072265625, 0.517578125, -0.97509765625, 0.52978515625, 0.14111328125, -1.4306640625, -0.55322265625, 0.95068359375, 1.552734375, 1.2861328125, 0.3779296875, 1.859375, -1.814453125, -0.6953125, -0.471923828125, 0.31787109375, 0.3828125, -0.087646484375, 1.361328125, 1.6953125, 1.5703125, -0.0963134765625, -1.4921875, 1.205078125, 0.38818359375, 1.3681640625, -0.67138671875, 0.491455078125, -0.7158203125, -1.2431640625, -0.436767578125, -0.95166015625, 0.1275634765625, 1.0576171875, -1.671875, 0.051116943359375, 1.8095703125, 0.0253143310546875, -1.40625, -0.12274169921875, 2.76171875, -0.5517578125, -0.41748046875, -0.89111328125, 0.8193359375, -3.068359375, -1.7548828125, 0.282470703125, -1.1181640625, 0.04510498046875, -0.95703125, 1.4716796875, 0.368896484375, 2.091796875, 0.189208984375, 1.048828125, 0.49853515625, 1.515625, -0.75, 1.779296875, -0.452880859375, -0.31982421875, -0.01351165771484375, 1.6875, -2.14453125, -0.2452392578125, -1.01171875, 0.951171875, -0.2548828125, -0.292724609375, -1.296875, -1.0849609375, 0.4609375, -0.7021484375, -1.2587890625, -1.01171875, -0.67724609375, -0.404541015625, -0.52490234375, 0.63134765625, 1.537109375, -0.8896484375, 0.23876953125, 0.234619140625, 0.353515625, -0.374755859375, 0.64306640625, 0.80029296875, 0.1148681640625, -0.63720703125, -0.52978515625, 0.256591796875, -1.5302734375, 0.380615234375, 0.2822265625, 0.60498046875, -1.3720703125, 0.2235107421875, -1.4365234375, 0.62841796875, -1.8955078125, -1.0947265625, 0.583984375, -0.81396484375, 1.0791015625, 0.11370849609375, -0.32177734375, 1.115234375, -0.2998046875, -2.037109375, 0.75048828125, -0.424560546875, 0.88916015625, -0.4404296875, 1.6083984375, 0.52099609375, -1.95703125, 0.11419677734375, -0.0233612060546875, 0.312255859375, 1.34375, -0.3671875, 1.931640625, -0.85009765625, -0.876953125, -1.658203125, -1.2900390625, -0.64990234375, 1.2353515625, -0.57568359375, 0.13525390625, 0.7001953125, 0.28076171875, 0.01263427734375, 1.1201171875, -0.1630859375, 0.90673828125, -0.08038330078125, 1.3076171875, -1.224609375, 0.047119140625, 0.94970703125, -2.662109375, 0.0217132568359375, 0.6328125, -0.490966796875, 0.57958984375, 0.127197265625, -0.623046875, -0.87451171875, 1.1689453125, -0.7763671875, -0.8828125, 0.3056640625, 1.6796875, -0.188232421875, 0.68603515625, 0.5498046875, -0.2457275390625, 1.193359375, -1.1640625, -1.27734375, 0.9580078125, 0.634765625, 1.7119140625, 0.77099609375, -0.1497802734375, -0.53173828125, 1.0, 0.0113677978515625, 1.609375, -1.255859375, -0.360595703125, -0.054595947265625, 0.3779296875, -0.417236328125, 0.0135955810546875, 0.2138671875, 1.0400390625, 0.12420654296875, 0.57861328125, -0.98486328125, 1.716796875, 1.8310546875, 0.0697021484375, 0.04345703125, -0.1981201171875, -0.386474609375, 2.943359375, 0.8828125, 1.6884765625, -0.202880859375, 1.3369140625, -1.703125, -1.087890625, -0.6328125, 0.96240234375, 0.77880859375, 1.4794921875, 1.1240234375, 0.07916259765625, -0.181884765625, 0.250732421875, -2.32421875, 0.331298828125, -1.8369140625, 0.04345703125, -1.2998046875, 1.1181640625, 0.1375732421875, 0.392578125, 0.7109375, -0.60205078125, 0.8564453125, -0.74853515625, 0.95947265625, 0.5615234375, -0.68798828125, 1.3349609375, 1.3515625, 0.78759765625, -0.3466796875, 1.3017578125, 0.69287109375, 1.3447265625, -0.381591796875, -2.826171875, -0.96142578125, 2.447265625, -0.7236328125, 0.477783203125, -2.328125, -0.53564453125, -0.398681640625, -0.4091796875, 0.619140625, -1.0791015625, -0.443115234375, -0.94921875, 0.353271484375, -0.9677734375, 0.302978515625, 0.85107421875, -0.41357421875, -1.5458984375, 0.1611328125, 1.2060546875, -0.65380859375, 0.560546875, 0.79052734375, 0.404052734375, 0.84375, -0.6572265625, 0.88720703125, -0.369384765625, -0.5908203125, 0.0972900390625, 0.666015625, -1.3037109375, 0.144287109375, 1.521484375, 0.393798828125, -0.85400390625, -1.8798828125, -0.74072265625, -1.4853515625, 0.12371826171875, -0.32763671875, -1.4775390625, 0.79833984375, -0.262451171875, 2.412109375, 0.0625, -0.294677734375, -1.2021484375, -0.96142578125, -0.26904296875, 0.3505859375, 1.4052734375, -0.361328125, -0.6318359375, -0.1807861328125, -0.791015625, 0.10223388671875, 0.2462158203125, -0.8193359375, 0.274169921875, 0.349853515625, -0.7158203125, -0.4306640625, -2.208984375, 0.96435546875, 1.833984375, -0.1488037109375, -1.4501953125, -1.234375, 1.173828125, -0.52099609375, -0.350341796875, 1.3515625, 0.908203125, -0.5791015625, -1.896484375, 0.869140625, 0.7255859375, 0.60888671875, -0.484375, -0.5224609375, 0.364501953125, -0.043212890625, -0.78759765625, 0.80810546875, -1.501953125, 0.25732421875, -0.49609375, 0.5712890625, -0.0712890625, -0.6474609375, 0.748046875, 1.361328125, -0.8916015625, 0.1973876953125, 1.0771484375, -0.49658203125, 0.1558837890625, -0.248779296875, 1.0771484375, -3.591796875, 0.82470703125, 2.224609375, -0.0260467529296875, -0.49072265625, 0.125732421875, -3.31640625, -0.89599609375, 0.12408447265625, 0.5771484375, 0.52197265625, 1.16796875, 0.8798828125, -1.7802734375, -0.2310791015625, -0.58837890625, 1.033203125, 1.818359375, -0.67138671875, -0.5244140625, -0.06549072265625, -0.293212890625, 0.2109375, 0.6552734375, -0.5908203125, -1.26171875, 0.921875, 0.64208984375, 1.0439453125, -0.1611328125, -1.12890625, 0.2137451171875, -0.91015625, 0.399658203125, -0.4482421875, -0.1424560546875, -0.24462890625, -0.7861328125, -1.3935546875, -0.83642578125, -0.09423828125, 0.90234375, -0.254150390625, -0.921875, 0.85693359375, 0.157470703125, 0.1876220703125, 1.7294921875, -1.1474609375, -1.1328125, 2.357421875, 0.9462890625, 0.3935546875, -1.1181640625, 1.6474609375, 1.369140625, -0.0254364013671875, -0.68212890625, 0.2080078125, -0.09197998046875, 0.361328125, 1.498046875, -1.337890625, 1.18359375, 1.0830078125, -1.7744140625, -1.384765625, 0.315185546875, -1.1435546875, 0.2449951171875, 0.5859375, -0.99169921875, 2.244140625, 0.35546875, -1.0888671875, 0.9365234375, -0.68310546875, 0.06817626953125, 0.8984375, -0.31494140625, 1.0673828125, 1.9443359375, 0.88818359375, -0.231201171875, 0.70361328125, -0.447998046875, -0.5791015625, -0.974609375, 0.37451171875, 1.509765625, -0.352294921875, 0.302734375, 1.8623046875, 0.7646484375, 0.27099609375, 1.267578125, 0.99365234375, 0.86376953125, -1.2177734375, -0.11328125, 0.8681640625, -0.943359375, 0.1317138671875, -0.423828125, -0.60546875, -0.0765380859375, -1.30859375, 0.485107421875, -1.07421875, 0.192626953125, -1.5556640625, -0.10968017578125, -0.85888671875, 0.421630859375, -1.16015625, 0.300048828125, 0.3369140625, 0.45263671875, -0.53515625, -0.6044921875, 1.0791015625, -0.9462890625, 0.607421875, 1.3349609375, 0.80322265625, -0.55810546875, 2.529296875, 1.1611328125, 0.82373046875, 0.61279296875, 0.58447265625, 1.0107421875, 0.58251953125, -1.630859375, 0.677734375, -0.0867919921875, 1.03515625, 0.59375, -0.81201171875, -0.41162109375, -1.1884765625, 0.51708984375, -1.43359375, 1.2998046875, 1.337890625, 1.251953125, -0.4765625, -1.2451171875, -0.10955810546875, 0.84814453125, 1.1357421875, -0.003772735595703125, -1.7275390625, -1.52734375, 1.4521484375, 1.32421875, 0.132568359375, 0.177734375, -2.302734375, 1.9482421875, 0.1414794921875, 0.53271484375, 0.0120086669921875, 1.9404296875, -0.46484375, 0.133544921875, 0.0836181640625, -0.9736328125, 0.11480712890625, 1.328125, -0.131591796875, -0.9697265625, -1.7001953125, 2.19921875, -1.255859375, -0.93505859375, 0.7431640625, 0.76513671875, -0.0068817138671875, -0.07025146484375, -0.939453125, 0.2103271484375, 1.7685546875, -0.55029296875, 0.429931640625, -0.70751953125, -1.1064453125, -1.4638671875, 0.48193359375, 0.51513671875, 1.4326171875, 0.82080078125, 0.89892578125, 1.1796875, 1.1533203125, 1.9453125, 0.5341796875, -0.272705078125, -0.273681640625, 1.9599609375, 0.0219573974609375, -0.830078125, -0.8837890625, 0.61962890625, -0.254150390625, 1.80859375, -2.033203125, 1.4541015625, 0.8095703125, -0.357421875, -0.52783203125, -0.96240234375, 0.1844482421875, 1.6572265625, -0.55517578125, -0.85205078125, -0.93798828125, -1.701171875, -0.462158203125, 1.5283203125, 0.080810546875, -0.6103515625, 0.0865478515625, -1.8330078125, 0.0518798828125, 0.392333984375, -0.482177734375, 0.552734375, 0.48046875, -0.15283203125, -0.4716796875, -1.458984375, 0.325927734375, -1.3310546875, -0.0255126953125, 1.08203125, -0.93798828125, -0.69287109375, 0.0302276611328125, -0.278076171875, 0.64453125, 0.0908203125, -0.5263671875, 0.1375732421875, 1.3291015625, -1.16796875, -0.1953125, -1.4794921875, 0.312744140625, -0.0016756057739257812, 0.9287109375, 0.2496337890625, 0.525390625, -2.173828125, -0.371826171875, 1.2529296875, 1.578125, -0.84423828125, 0.15673828125, 0.06964111328125, 0.4052734375, 1.1484375, -1.1513671875, -1.0703125, 1.1884765625, 0.70263671875, -1.439453125, -0.9794921875, 0.250732421875, -0.36865234375, 0.3642578125, -0.08184814453125, 1.130859375, 0.23486328125, 1.3115234375, -0.814453125, -0.1392822265625, -0.77880859375, -0.99853515625, -1.2783203125, 0.80517578125, 0.285400390625, 0.58837890625, -0.2020263671875, 0.958984375, -0.382568359375, -0.03314208984375, -1.205078125, -0.9794921875, -0.270751953125, -0.1431884765625, -0.041015625, -0.376220703125, -1.869140625, 2.068359375, 1.765625, -1.8837890625, -0.01108551025390625, 0.371826171875, -1.3896484375, -0.137939453125, 0.154052734375, -0.86865234375, -0.27880859375, -0.72998046875, -2.2734375, 1.5615234375, -0.318359375, 0.141357421875, -0.006160736083984375, -0.75244140625, -0.79443359375, -0.6259765625, -0.88134765625, -1.0244140625, 0.9892578125, -0.285400390625, -0.433837890625, -1.10546875, 0.763671875, -1.7705078125, 1.4541015625, 0.161865234375, 0.88916015625, -2.029296875, 0.1942138671875, -1.58203125, -1.7783203125, 1.8505859375, -0.1673583984375, 0.63232421875, -1.0712890625, 0.2103271484375, 2.1796875, -0.5244140625, -0.419677734375, -0.07855224609375, -1.232421875, 1.4296875, -1.2373046875, 1.1953125, -0.26171875, 2.26171875, 0.343017578125, 1.4541015625, 1.4775390625, -0.2142333984375, 0.7587890625, 0.64892578125, 0.306884765625, -0.61328125, 1.5751953125, 0.076171875, -1.0224609375, 1.0185546875, 0.413818359375, 1.6455078125, -0.2000732421875, -2.689453125, -0.041717529296875, -0.7265625, -0.403564453125, -1.40625, -0.2138671875, 0.335693359375, 0.53173828125, 0.5634765625, -0.9013671875, -1.07421875, 1.365234375, 0.428955078125, -1.3681640625, 0.6884765625, 0.84912109375, -1.2529296875, -0.578125, -0.74560546875, -0.96826171875, 0.0123443603515625, 0.8984375, 1.2919921875, 1.32421875, 1.4111328125, -0.08721923828125, 0.197021484375, 0.7041015625, -0.11151123046875, -0.9931640625, -0.174072265625, 1.54296875, -1.173828125, -0.181640625, 1.3857421875, -1.076171875, 1.19140625, 1.173828125, -0.93408203125, 1.296875, -0.1759033203125, -0.02838134765625, -1.369140625, 1.46484375, -1.12109375, -0.435546875, 1.59765625, -0.61328125, 1.212890625, 0.1517333984375, -0.0028133392333984375, -0.033233642578125, 0.476806640625, 2.5859375, -0.57958984375, -1.3193359375, -0.63427734375, 1.1689453125, 0.1319580078125, 1.314453125, 1.458984375, -1.1865234375, 1.15234375, -1.513671875, -0.01568603515625, 0.9716796875, -0.0738525390625, 0.822265625, -1.0595703125, -0.62353515625, -0.375244140625, 0.313232421875, 0.394775390625, 0.0555419921875, 0.10089111328125, 1.2109375, -0.44921875, -1.8974609375, 0.6044921875, 0.630859375, 0.255859375, -1.8720703125, -0.63134765625, 0.04669189453125, -1.0185546875, 0.478515625, 1.794921875, -0.31982421875, -0.2330322265625, -1.2431640625, -4.0531158447265625e-05, -0.06817626953125, 0.09735107421875, -0.095458984375, -0.2049560546875, 0.2386474609375, -0.419677734375, 1.2841796875, -0.44287109375, 0.2322998046875, -0.8125, 2.013671875, -0.658203125, 0.297607421875, 0.6728515625, 1.185546875, 1.388671875, -0.71728515625, 1.072265625, -1.7568359375, -0.06793212890625, -0.64990234375, -0.1051025390625, -1.416015625, -1.6474609375, -2.365234375, 0.174072265625, -0.6357421875, -0.47802734375, -0.26318359375, -0.0146331787109375, -1.5302734375, 0.59912109375, -0.9052734375, -0.63037109375, 1.240234375, 0.0185546875, 0.1646728515625, 0.264892578125, 0.82373046875, 1.3173828125, -0.95263671875, 0.1231689453125, -0.8173828125, 0.666015625, -0.31787109375, -0.7822265625, -0.51708984375, -0.6435546875, -1.6904296875, 0.392578125, 0.476318359375, -0.312255859375, -0.387451171875, 0.0919189453125, 1.146484375, -1.3916015625, 0.452880859375, 0.25390625, 0.55078125, -0.087646484375, 0.28466796875, 0.0577392578125, -0.2476806640625, -1.2451171875, -0.65673828125, 0.364013671875, -0.449951171875, 0.08831787109375, -0.4345703125, 0.374267578125, 0.08013916015625, 1.0712890625, -0.271484375, 1.65234375, -1.255859375, 1.751953125, -1.603515625, 0.27880859375, 0.47509765625, -0.736328125, 1.22265625, -0.2164306640625, 1.0419921875, 0.58056640625, 0.14599609375, -0.275634765625, -1.2646484375, 0.7353515625, 2.40234375, -0.307373046875, 1.0546875, -0.460205078125, 1.4306640625, -0.435791015625, -0.62060546875, 0.591796875, -1.1767578125, 1.6552734375, -0.35986328125, -0.55126953125, 2.369140625, 0.72900390625, -0.369384765625, -1.125, -0.178955078125, -0.638671875, 0.11859130859375, 0.280517578125, -0.1790771484375, -1.5810546875, -1.0361328125, -0.6728515625, -1.1845703125, 1.130859375, 0.2783203125, -0.285888671875, -0.62060546875, 0.6220703125, 0.82177734375, 0.92626953125, 2.365234375, 1.6181640625, -0.0266876220703125, -0.6640625, -0.021697998046875, -0.387451171875, 1.060546875, -1.65625, -0.55615234375, -0.032501220703125, -0.3115234375, 1.66015625, 0.2139892578125, 0.52001953125, 0.6669921875, -0.0107269287109375, 1.1435546875, 1.681640625, 0.83642578125, -0.27392578125, -2.095703125, -1.5966796875, -1.6279296875, 1.3310546875, -0.61865234375, -0.423095703125, -0.7255859375, 0.2481689453125, 2.16015625, -1.234375, -2.068359375, 1.1181640625, 0.08624267578125, -1.6015625, -1.1845703125, 0.1455078125, -1.7255859375, 0.61083984375, 0.88330078125, -0.427490234375, 0.345458984375, 0.779296875, 1.798828125, 0.463623046875, 0.06732177734375, 1.8916015625, -0.2247314453125, 1.34765625, -0.431884765625, -0.2061767578125, -2.2734375, 0.3125, 1.853515625, 1.150390625, -1.15234375, -0.58203125, -0.63232421875, -0.859375, -0.12249755859375, 0.217041015625, -0.29296875, -0.11572265625, -0.91259765625, -0.186279296875, -0.84375, -1.3583984375, 0.63134765625, 0.29541015625, -0.5234375, 1.1162109375, 0.417236328125, 1.111328125, -1.53515625, -1.775390625, -0.2462158203125, 0.87255859375, 0.1302490234375, 0.6201171875, -0.6767578125, -1.6640625, -1.2724609375, 0.2332763671875, -0.6650390625, 0.6513671875, -1.94140625, -0.44873046875, 0.254638671875, -0.951171875, -0.7763671875, 0.2337646484375, -0.222412109375, 0.419677734375, 0.6484375, 1.080078125, 0.3896484375, 0.64013671875, -0.8154296875, -1.44921875, 1.2939453125, 0.12249755859375, 0.0867919921875, -1.77734375, 1.7041015625, -1.2138671875, -1.365234375, -0.220703125, -0.62890625, 0.7509765625, -1.30859375, -0.80322265625, -0.2578125, 0.00872802734375, -0.278564453125, -0.537109375, -1.4248046875, 1.6083984375, 0.6572265625, -2.037109375, -0.0035858154296875, -1.169921875, 0.09161376953125, -0.2083740234375, 1.099609375, 2.078125, 0.1746826171875, 0.8046875, 1.02734375, 0.1787109375, -1.271484375, 0.10540771484375, -0.454833984375, 0.5888671875, -0.54345703125, 0.890625, 0.186767578125, -0.8798828125, 0.2119140625, -1.7919921875, 0.65771484375, -0.433837890625, 0.7080078125, -0.8369140625, 0.468017578125, 1.0791015625, -0.8232421875, -0.7822265625, -0.45556640625, -0.3447265625, 0.85400390625, 0.54638671875, -0.544921875, 0.25439453125, -1.55859375, -0.85693359375, 0.6103515625, 0.1962890625, 0.1533203125, -0.802734375, -0.466796875, -0.98828125, 0.51953125, -0.2783203125, 0.1988525390625, -0.485107421875, 1.1123046875, 0.396728515625, -1.2021484375, 0.376708984375, 0.978515625, -0.287353515625, 0.0943603515625, 0.0985107421875, 0.08673095703125, 0.165771484375, 0.6162109375, 0.003467559814453125, -1.6044921875, 0.900390625, -0.96044921875, -0.263427734375, -0.049896240234375, 1.4140625, -0.8623046875, -0.6875, 0.3232421875, 0.310302734375, 0.7880859375, 0.2281494140625, 0.272216796875, 0.2080078125, 0.0038700103759765625, -0.0235443115234375, -0.11639404296875, -0.79833984375, -1.0810546875, 0.460693359375, 0.295654296875, 0.08282470703125, 1.9951171875, -1.1904296875, -1.75390625, -0.712890625, -0.058837890625, -0.2203369140625, 1.1259765625, 0.9326171875, -1.5751953125, -1.9892578125, 1.015625, 0.1329345703125, 1.23828125, 1.466796875, 0.075927734375, -0.77490234375, 1.1591796875, -0.5244140625, 1.0048828125, -1.06640625, -1.2890625, -0.57568359375, 0.40478515625, 0.91845703125, -0.314697265625, -0.86181640625, -2.033203125, -0.250732421875, -0.8076171875, 0.031494140625, -0.85302734375, -1.4814453125, 0.02752685546875, 1.1669921875, 0.1622314453125, 0.15087890625, 0.06597900390625, -0.7255859375, -0.087646484375, -1.8740234375, -0.282958984375, -0.326416015625, -1.20703125, -0.49267578125, -0.84375, 1.8505859375, -1.953125, -0.80224609375, -0.5244140625, -0.826171875, 0.466552734375, 1.12890625, 1.1904296875, 1.9736328125, 1.00390625, -1.15234375, 0.5029296875, -0.89404296875, -0.379638671875, -0.755859375, 0.137451171875, 1.3251953125, -0.22900390625, 0.270263671875, 0.11102294921875, -1.1376953125, -1.28515625, 1.1650390625, -0.6181640625, 0.054473876953125, 0.98095703125, -1.2138671875, 1.0166015625, -0.59326171875, -0.6728515625, 0.464111328125, -0.281982421875, -0.98681640625, 0.91845703125, -0.0841064453125, 0.19921875, 0.51904296875, -0.1412353515625, -0.54736328125, -0.869140625, 1.7412109375, 1.390625, 0.0653076171875, -2.353515625, -1.2197265625, 0.107666015625, -1.775390625, -0.344482421875, 0.8037109375, 0.352294921875, 0.168212890625, 1.5810546875, 0.69873046875, 0.27734375, -0.67431640625, -0.250244140625, -0.4521484375, 0.71728515625, -0.2464599609375, -1.7314453125, -1.224609375, -0.6494140625, 1.80859375, -0.264892578125, -2.8125, 1.0888671875, 0.4755859375, 0.2119140625, -2.517578125, 0.63671875, -0.59814453125, 0.2353515625, 0.78125, -0.316650390625, -0.56103515625, -0.2156982421875, -0.57666015625, -0.78369140625, -0.6416015625, -0.52783203125, -0.27099609375, 1.65625, -1.0634765625, -0.212646484375, -0.544921875, 0.5244140625, -1.5673828125, -1.3408203125, 0.265625, -0.038909912109375, 0.97021484375, 1.265625, 0.407470703125, 1.61328125, 1.3056640625, -0.440185546875, 0.2479248046875, -0.10400390625, 1.494140625, 0.72900390625, -0.810546875, -0.99072265625, -1.0576171875, 0.4990234375, -0.93994140625, -1.0751953125, -0.7021484375, -0.9462890625, -1.0302734375, -0.7890625, -0.4912109375, 0.295654296875, 0.73828125, -0.640625, -0.18017578125, 1.44140625, -0.375244140625, 0.124267578125, -0.08221435546875, -0.7900390625, 1.982421875, -0.226806640625, 0.05548095703125, -0.284423828125, -1.0107421875, -1.005859375, -0.474853515625, 0.32373046875, 0.042938232421875, -0.12841796875, 1.1025390625, 0.4716796875, 1.58203125, -0.40771484375, 1.6875, 1.9794921875, 0.33740234375, -0.4697265625, -0.1123046875, 0.189697265625, -0.497802734375, 0.406494140625, 0.07989501953125, -1.6025390625, 0.81591796875, -1.0869140625, 0.1473388671875, 1.94921875, -0.7548828125, 0.09893798828125, -0.361572265625, -1.087890625, 0.4296875, 0.0267181396484375, 2.3671875, 1.26171875, -0.59228515625, -0.19482421875, -1.033203125, 0.37548828125, -0.2406005859375, -0.42333984375, -0.8876953125, 1.4853515625, -0.07049560546875, 0.078857421875, 0.045806884765625, 2.138671875, -0.441162109375, -0.376220703125, 0.83837890625, -0.267578125, -0.2435302734375, 0.68310546875, -0.08587646484375, 1.263671875, 0.80078125, -0.69580078125, -1.05859375, -0.011138916015625, 0.222412109375, -1.0400390625, -1.234375, -0.07391357421875, 0.70556640625, -0.5458984375, -0.037872314453125, -2.2421875, -0.7451171875, 1.0341796875, 1.19140625, -0.41015625, -0.1275634765625, -0.434814453125, -1.0517578125, -0.79833984375, 0.1949462890625, 0.6728515625, -0.339111328125, -1.275390625, 0.5732421875, 0.32373046875, -1.0673828125, -0.190673828125, -0.82373046875, -0.393310546875, -1.12890625, 0.7978515625, -0.9931640625, -0.259765625, 0.51806640625, 0.272705078125, 0.67529296875, 2.6328125, 0.302001953125, 1.7626953125, 0.307861328125, 1.08984375, -0.364013671875, -1.828125, -0.52001953125, 0.35986328125, -1.5712890625, -0.07208251953125, -0.6044921875, 1.6171875, 0.8681640625, -0.135498046875, 1.658203125, -0.18896484375, -1.828125, 0.9140625, -0.365478515625, 0.2296142578125, 1.49609375, 1.4521484375, 0.34423828125, 1.5771484375, -0.49853515625, -0.8330078125, 0.10980224609375, 2.580078125, -0.486083984375, 1.3388671875, -1.4072265625, -1.4287109375, 1.2041015625, -1.4697265625, 0.97119140625, -1.13671875, 0.442138671875, 0.08514404296875, -0.149169921875, -0.1627197265625, 0.640625, -0.44873046875, 2.224609375, -0.44287109375, 0.5126953125, -0.2147216796875, 0.142578125, 0.82421875, 0.2734375, -0.451904296875, 0.1785888671875, 0.473876953125, -0.76708984375, 1.62890625, 0.36669921875, 0.1905517578125, 0.2022705078125, 0.697265625, -0.9013671875, -0.64306640625, 0.69970703125, 0.853515625, -0.5419921875, -0.07379150390625, -0.298583984375, 0.50048828125, 0.93310546875, -0.51220703125, -0.0926513671875, 0.463134765625, 0.37744140625, -0.86181640625, 0.5908203125, 0.27685546875, -0.63330078125, 0.513671875, 0.72265625, 0.64404296875, 0.1845703125, 0.2430419921875, 0.8408203125, 1.46875, 0.56689453125, -0.822265625, 0.33642578125, -1.0322265625, 0.52294921875, -0.75, -0.38134765625, -0.9482421875, -0.61181640625, 1.0546875, 1.1982421875, -1.1572265625, -0.79736328125, 1.7080078125, -0.1654052734375, 0.8388671875, 0.732421875, -0.81201171875, 0.619140625, 0.385498046875, -1.861328125, -2.880859375, -0.6953125, 1.4091796875, 0.837890625, 0.1312255859375, 0.6064453125, 0.26513671875, 0.2376708984375, -1.234375, 0.95751953125, 0.66552734375, 0.52587890625, -0.160888671875, -1.2939453125, -1.52734375, -0.298828125, 0.57568359375, 0.8408203125, -0.2666015625, 0.99267578125, -1.384765625, 1.4765625, -1.9970703125, -0.46728515625, 1.7919921875, -0.30810546875, -1.3505859375, -0.255859375, 0.01076507568359375, 0.425048828125, 2.4375, 0.59375, 1.091796875, -0.66650390625, -0.3466796875, -0.445068359375, 2.3125, 0.388916015625, 0.95361328125, -1.0712890625, -0.4521484375, 0.76416015625, -0.548828125, -0.94482421875, 0.712890625, 1.162109375, -0.031707763671875, 2.390625, -0.91455078125, 0.7685546875, -0.265625, -0.43310546875, -0.392578125, 0.1748046875, 0.2890625, 0.94091796875, -0.11328125, -0.4599609375, 0.482421875, 1.2734375, -0.724609375, 0.81689453125, 0.263427734375, 1.0947265625, 0.386962890625, 0.08001708984375, 0.2301025390625, 0.52490234375, -0.190673828125, -0.64892578125, 0.09613037109375, -1.3525390625, -1.3251953125, -1.4736328125, 0.93017578125, 1.4755859375, -0.1864013671875, -0.763671875, 0.91455078125, 1.0234375, 0.6025390625, 1.9765625, -1.244140625, 0.14501953125, 1.04296875, 0.7822265625, 1.087890625, -0.1595458984375, 0.3193359375, 0.6123046875, 0.2388916015625, -1.2119140625, -1.1044921875, -0.79541015625, -0.301513671875, -0.071044921875, 2.4375, 0.521484375, 1.126953125, 1.076171875, -0.900390625, 1.8173828125, 0.96923828125, 1.615234375, 1.392578125, 0.202880859375, -0.1502685546875, -2.2109375, -0.86376953125, -0.48046875, 0.06378173828125, 0.54248046875, 0.055328369140625, -1.83203125, -0.3525390625, 0.46923828125, -1.0283203125, 1.576171875, -0.079833984375, -1.0, 0.89990234375, 0.71630859375, 0.046875, -0.615234375, 1.5703125, 0.7978515625, 0.11273193359375, 1.0126953125, -0.044097900390625, -1.7451171875, -0.64599609375, -1.2236328125, 0.6767578125, 0.1444091796875, -0.39990234375, -1.3955078125, -2.310546875, 1.560546875, 0.5478515625, 0.188232421875, -0.315673828125, -0.06402587890625, 0.1416015625, 0.64111328125, 0.06805419921875, 1.5595703125, -0.010772705078125, 0.07696533203125, -1.431640625, 0.2054443359375, 0.046600341796875, -0.1336669921875, 0.69580078125, 1.40625, -0.078369140625, 1.955078125, 0.467041015625, 0.4658203125, 0.8046875, 0.1444091796875, -0.1788330078125, 0.199462890625, 2.2578125, -1.2041015625, 0.34619140625, 0.2266845703125, 1.30859375, -0.8330078125, -0.732421875, 0.7080078125, -1.259765625, 1.751953125, 0.765625, 0.207763671875, 0.1885986328125, 0.2264404296875, 2.169921875, 0.4833984375, -0.1759033203125, 0.035491943359375, -0.06207275390625, -0.5986328125, 0.488037109375, -0.98876953125, -0.677734375, -1.4560546875, -0.5498046875, -0.17041015625, -0.334716796875, -1.8564453125, -0.6044921875, -0.51513671875, -1.2724609375, 1.009765625, 1.4541015625, -0.46435546875, -1.7333984375, -0.6708984375, 0.56591796875, 0.86474609375, 0.5634765625, 0.8115234375, 0.448486328125, -0.23388671875, -0.89501953125, 1.0009765625, 0.4443359375, -1.2138671875, -0.544921875, -0.420654296875, 1.69921875, -0.0504150390625, 0.71875, -2.556640625, 0.5498046875, -0.137939453125, 0.358642578125, 0.71875, -0.54443359375, 0.380126953125, 0.86376953125, -0.09442138671875, -1.4560546875, -1.0068359375, -1.501953125, -0.814453125, 0.2308349609375, 0.8974609375, 0.85205078125, -0.1727294921875, -0.7919921875, -0.6201171875, -0.07305908203125, -1.677734375, -2.34765625, 0.4013671875, -1.2666015625, -1.2060546875, -0.169189453125, -0.2393798828125, -0.7666015625, -1.7705078125, 0.78173828125, -1.404296875, -1.63671875, 0.4765625, 0.5537109375, 0.404541015625, -0.234619140625, 0.61572265625, 1.109375, -0.2347412109375, 2.08984375, -0.7412109375, -0.064697265625, -2.064453125, -0.11370849609375, -0.0036792755126953125, -0.99755859375, 1.044921875, -1.095703125, -0.17578125, -1.205078125, 0.1766357421875, -0.205810546875, 0.03912353515625, 1.3984375, 0.48291015625, 0.20849609375, -0.80419921875, -0.91455078125, 0.256591796875, -0.8330078125, -0.2314453125, -2.38671875, 1.1611328125, 0.120361328125, -0.259765625, 1.416015625, 0.384765625, -0.50390625, 0.27978515625, -0.7392578125, 2.072265625, -0.58837890625, -0.0018110275268554688, -0.288330078125, 0.83154296875, 0.2154541015625, -0.7626953125, -0.2401123046875, 1.2939453125, 0.2166748046875, -0.173095703125, -0.330322265625, 1.0634765625, 1.0205078125, 1.427734375, -0.77587890625, -0.083984375, 0.23193359375, 1.1123046875, 0.1866455078125, -1.080078125, 0.5576171875, -0.23291015625, -1.478515625, 0.701171875, -0.8056640625, 0.31591796875, 0.342529296875, -1.5732421875, 1.8642578125, 0.306396484375, 1.1171875, 0.10015869140625, -1.404296875, -0.3740234375, -0.9609375, 0.048248291015625, -0.296875, -0.05352783203125, 1.15234375, 0.966796875, -0.6298828125, 0.39599609375, 0.49853515625, 1.201171875, 0.352783203125, -1.15234375, -1.099609375, 1.412109375, 0.63134765625, 0.20166015625, -0.482177734375, -1.369140625, 0.4677734375, -0.91162109375, 0.432373046875, -0.427001953125, -0.27978515625, -1.0322265625, -0.587890625, -1.052734375, -0.935546875, -1.3837890625, -0.248291015625, 1.3388671875, -1.4267578125, 0.87939453125, 0.98583984375, -1.755859375, 0.40234375, -0.7177734375, -1.56640625, -1.20703125, 0.2822265625, 0.5693359375, -0.39453125, 0.460205078125, -0.007171630859375, 0.11920166015625, 1.1796875, -0.03326416015625, 2.642578125, -1.0517578125, -1.3759765625, -1.580078125, -0.146728515625, 0.6640625, 1.9580078125, -0.53369140625, 1.1962890625, -0.92529296875, 0.427978515625, -1.4912109375, 0.0135040283203125, -1.673828125, 2.43359375, 0.87744140625, -1.9150390625, -0.00820159912109375, -0.759765625, -0.50390625, -0.2587890625, -0.79345703125, -0.2049560546875, 0.58447265625, -0.41162109375, -1.2197265625, 0.2452392578125, -0.83935546875, 0.11126708984375, 0.80322265625, 1.0263671875, 0.2388916015625, -0.2120361328125, 2.02734375, -0.299560546875, 0.44921875, -0.316650390625, -1.9755859375, -0.329833984375, -0.1741943359375, 0.85595703125, 1.2099609375, 0.261474609375, -0.28173828125, -1.4765625, 0.468994140625, -0.494384765625, 0.252197265625, 0.8466796875, -0.70947265625, -0.6875, -0.333984375, 0.6337890625, -0.62890625, -0.8359375, -0.88525390625, 1.0234375, 1.3134765625, -0.93359375, 0.355224609375, -0.391845703125, 0.0213623046875, 1.53125, -1.0224609375, -0.38525390625, 0.64208984375, -0.16650390625, -0.130126953125, -1.31640625, 1.96875, -1.404296875, -0.69873046875, 1.552734375, 0.57568359375, -0.1700439453125, -0.884765625, 0.09954833984375, -0.1629638671875, -0.40234375, 0.1103515625, 0.417236328125, -1.44140625, 0.491455078125, -0.2021484375, 1.2314453125, -0.67724609375, 0.92578125, 0.59765625, 0.16552734375, -0.35546875, 0.9462890625, -0.28369140625, 1.115234375, 1.212890625, -0.09295654296875, 0.457763671875, -1.419921875, -0.359130859375, -0.67529296875, 0.2298583984375, -0.330810546875, -1.123046875, -0.84716796875, 0.0836181640625, 0.374267578125, 0.50244140625, 1.21875, 1.033203125, -1.57421875, 1.8134765625, -1.1220703125, -0.63916015625, -0.87646484375, -0.3984375, -1.125, -0.485107421875, 0.3798828125, 0.79052734375, 1.1123046875, 0.6328125, 0.669921875, -1.642578125, 0.1617431640625, 1.236328125, 0.010101318359375, -0.419921875, 1.2802734375, -0.4404296875, 0.58203125, -0.30908203125, -0.373046875, 0.299560546875, -0.49951171875, 1.0654296875, 0.049774169921875, 1.0048828125, 0.677734375, -0.224609375, 0.1265869140625, -0.82421875, 0.9169921875, -0.375244140625, 0.92333984375, 0.9287109375, 0.434326171875, 0.1773681640625, 0.67724609375, 1.271484375, 1.0185546875, 0.83984375, -1.5625, -0.097900390625, -0.572265625, -0.1488037109375, 0.1258544921875, -0.2841796875, -0.282958984375, -0.0736083984375, -1.31640625, -1.412109375, 0.7724609375, -0.630859375, -2.171875, -0.416748046875, 2.333984375, 0.9130859375, 0.216552734375, 0.7490234375, -0.461181640625, 1.1630859375, -1.236328125, -0.54296875, 1.951171875, -0.58447265625, 1.7060546875, 0.05126953125, 1.283203125, 0.8310546875, 0.3974609375, 1.6884765625, -0.75537109375, -0.2130126953125, -0.36669921875, -0.00701141357421875, -0.60400390625, 1.3525390625, -1.7333984375, -1.4609375, -2.29296875, 0.9296875, 1.22265625, -0.212646484375, -1.6923828125, 0.67333984375, -0.046844482421875, -0.472412109375, 0.611328125, -0.471435546875, 0.491455078125, 0.270751953125, -0.8427734375, -1.03515625, -1.8369140625, -0.1510009765625, 0.00424957275390625, 1.962890625, 0.19482421875, 0.900390625, -0.003082275390625, -1.54296875, -1.33203125, -1.3701171875, -0.66455078125, -0.642578125, 1.0439453125, -0.036285400390625, 1.9345703125, -0.9521484375, 1.578125, -0.45361328125, -0.56689453125, -0.931640625, -1.345703125, 1.2333984375, 0.10455322265625, 1.4111328125, -0.1744384765625, -0.7001953125, 1.0517578125, 1.37109375, 0.49267578125, 0.546875, 0.388916015625, 1.021484375, -1.0947265625, 0.73779296875, -0.3330078125, 0.33740234375, -0.486083984375, 0.0491943359375, 0.320556640625, 0.446044921875, 2.041015625, 0.73779296875, 0.081787109375, 0.84033203125, -0.212158203125, 2.154296875, -1.1552734375, -0.1448974609375, 0.05517578125, -0.029693603515625, -0.732421875, -0.221435546875, -0.245361328125, -0.5009765625, 0.1488037109375, 0.91943359375, 0.544921875, 0.11944580078125, -0.1474609375, -0.5869140625, 1.02734375, -0.378173828125, -0.7998046875, -0.460205078125, -1.4375, -1.0263671875, 1.048828125, -0.54150390625, 0.80810546875, -1.162109375, -1.048828125, -0.30810546875, 0.6005859375, 0.0638427734375, 0.802734375, 0.9501953125, 0.07843017578125, 0.65771484375, -0.36083984375, -0.25439453125, 0.33447265625, 0.28369140625, 1.048828125, -1.3818359375, -0.29736328125, 0.86279296875, -1.1962890625, -0.1458740234375, -0.7021484375, -0.95751953125, 1.2451171875, 0.42333984375, 1.6103515625, -1.376953125, -2.208984375, 0.267822265625, 0.43603515625, -0.26953125, 1.486328125, -0.7568359375, 0.033966064453125, -0.0972900390625, 0.818359375, 0.224853515625, 0.2249755859375, 0.5849609375, 1.060546875, 0.2388916015625, -1.4404296875, 0.58984375, -0.6298828125, 0.151123046875, 1.6416015625, 0.08282470703125, 1.1630859375, 0.287841796875, -1.2392578125, -0.3935546875, 1.609375, 0.346435546875, -0.072998046875, -0.56640625, 0.363525390625, 0.7470703125, 2.626953125, 0.34912109375, -0.62255859375, 0.276611328125, 1.3935546875, 0.3203125, 0.927734375, -0.09423828125, 0.2286376953125, 0.430908203125, -0.54638671875, 0.52197265625, -1.7548828125, 0.7822265625, 0.31640625, 0.442626953125, 1.056640625, -0.74609375, -1.15234375, -0.93603515625, -1.4091796875, -0.357421875, 1.19921875, -0.1435546875, -1.46875, 0.67919921875, -0.60107421875, 1.4951171875, 0.015960693359375, -0.44775390625, 0.285400390625, 1.458984375, 0.5234375, -1.3623046875, -0.8984375, 0.65283203125, 1.43359375, 0.05657958984375, 0.40966796875, 0.91455078125, -2.248046875, 0.84619140625, -1.4990234375, 1.2080078125, -0.414306640625, 0.270751953125, 0.666015625, 1.052734375, 1.9462890625, 1.5322265625, 1.357421875, 0.07421875, -1.115234375, -0.35693359375, -2.158203125, -1.0732421875, -0.040191650390625, -0.157958984375, -1.5029296875, -0.7646484375, 0.0188751220703125, 1.54296875, -0.66259765625, 0.8095703125, -1.1943359375, 0.76025390625, 0.077880859375, -0.273193359375, 0.11761474609375, -0.61767578125, 0.724609375, 1.296875, 1.04296875, -0.20849609375, 0.05523681640625, 2.5234375, -0.8681640625, -0.076171875, -0.57568359375, -0.2393798828125, 0.54443359375, -0.03900146484375, 2.015625, -1.5107421875, 0.484619140625, 0.6826171875, 0.106689453125, 0.1571044921875, -0.5380859375, 1.1552734375, 0.52197265625, -0.441650390625, 0.208984375, -1.146484375, -0.0175628662109375, 1.87109375, 0.6357421875, 0.25146484375, -0.2264404296875, 0.0706787109375, 0.364013671875, -1.677734375, 1.775390625, -1.1396484375, 0.615234375, -0.98193359375, -0.325439453125, 1.263671875, 0.740234375, -1.626953125, -1.560546875, 0.157470703125, -0.5458984375, -0.7314453125, 1.3955078125, -0.401123046875, 0.49951171875, -1.57421875, -1.4951171875, -0.235107421875, 0.209716796875, 1.462890625, 0.86962890625, -0.7333984375, 0.01364898681640625, -2.181640625, 1.0478515625, 0.607421875, -0.70361328125, 0.697265625, -1.474609375, 0.039459228515625, -0.259033203125, -1.2880859375, 0.412841796875, -0.040283203125, -1.38671875, -0.88037109375, -0.91064453125, -0.419677734375, 0.865234375, 0.37744140625, 0.34423828125, -0.6650390625, -1.322265625, 0.2435302734375, -1.5537109375, 0.84814453125, 1.6298828125, 0.1746826171875, -0.66455078125, 0.436767578125, -2.16015625, 0.1917724609375, -3.359375, 0.094482421875, -0.5390625, -0.046234130859375, -1.2587890625, 0.80908203125, 1.2216796875, -0.90087890625, -0.384033203125, 0.033721923828125, 1.9384765625, -0.422119140625, 1.494140625, -1.841796875, -0.12188720703125, -1.4306640625, -0.27001953125, 0.11785888671875, -0.51708984375, -0.30810546875, 1.4443359375, 0.86181640625, -1.0107421875, 0.451904296875, -0.489013671875, -0.763671875, 0.57470703125, -0.6552734375, -0.64111328125, -1.1845703125, 0.646484375, 0.40966796875, 0.70849609375, 0.06353759765625, -0.966796875, -0.62353515625, -0.2127685546875, -1.4033203125, -1.3310546875, -1.234375, -0.51513671875, -0.33935546875, -0.6640625, -0.73193359375, 0.5341796875, -2.93359375, -0.453125, -0.2958984375, -1.7734375, -1.22265625, -1.10546875, -1.525390625, 1.6396484375, 0.27587890625, -0.0662841796875, 1.0087890625, 0.76318359375, -0.4580078125, 0.5439453125, 1.3837890625, 1.7421875, -0.10675048828125, -0.95458984375, 2.55859375, -0.80712890625, 0.1483154296875, -0.95068359375, -0.755859375, 0.525390625, -0.35107421875, 1.2333984375, 0.65087890625, -2.513671875, -0.0875244140625, -0.2274169921875, 0.8115234375, 0.47412109375, 1.76953125, -1.4404296875, -0.8173828125, 0.0330810546875, -0.1392822265625, -0.043975830078125, -0.367431640625, -1.025390625, -0.896484375, 1.720703125, 1.3154296875, -1.0712890625, -1.3369140625, 0.5126953125, -0.6669921875, 0.1318359375, -0.7431640625, 0.6826171875, 0.60205078125, 0.06097412109375, 1.3271484375, -0.3505859375, 1.6826171875, -1.03515625, 0.68896484375, 1.4736328125, -1.0380859375, -2.455078125, 0.7509765625, 0.830078125, 1.625, 0.64208984375, -1.45703125, -0.485595703125, -0.97900390625, -1.18359375, -2.201171875, 1.5458984375, 0.99365234375, -0.90087890625, 0.68798828125, 0.7197265625, -0.327880859375, 1.3359375, 0.48876953125, 0.12127685546875, 1.2666015625, -0.139892578125, -0.83154296875, 1.349609375, -0.5927734375, -1.080078125, 0.958984375, 0.9384765625, -1.37109375, 1.4052734375, 1.4150390625, -0.4619140625, -0.990234375, -1.9716796875, 1.3818359375, 2.19140625, 0.2310791015625, -0.9208984375, 0.417236328125, 1.28515625, 0.9619140625, -0.474853515625, 0.76708984375, 1.2109375, 0.130126953125, -0.08056640625, 1.736328125, 0.405029296875, -0.42626953125, -1.8076171875, 0.5830078125, -1.0498046875, 1.234375, 0.615234375, -1.9072265625, 0.0020465850830078125, -0.55810546875, 0.8408203125, 0.98828125, 0.411865234375, -0.25341796875, 0.73046875, 1.30859375, 1.2470703125, 0.697265625, 0.89111328125, -0.6611328125, -0.51953125, 1.9365234375, 0.76123046875, -0.111328125, 0.457275390625, -1.248046875, 0.37060546875, 0.37060546875, 1.259765625, -1.4599609375, -1.4892578125, -0.70849609375, -0.1298828125, -0.110595703125, -0.86474609375, -0.1597900390625, 0.36328125, -0.058441162109375, 0.2230224609375, 1.5986328125, -0.5947265625, 0.2349853515625, -0.149169921875, -1.154296875, 0.9052734375, 0.373779296875, 0.24072265625, 0.6591796875, 0.45166015625, 0.08526611328125, 1.998046875, 1.8603515625, 0.056304931640625, -0.12091064453125, -0.69921875, -0.4150390625, -0.022857666015625, -1.0966796875, -0.65869140625, -0.1956787109375, 0.98095703125, 1.1630859375, -0.207763671875, 0.646484375, 0.87890625, 1.1142578125, 0.5107421875, -1.8505859375, -0.088623046875, -0.359619140625, 1.34375, -0.99755859375, 2.263671875, 0.28759765625, -0.12139892578125, -0.037872314453125, -0.7998046875, 1.0009765625, 1.85546875, 0.397216796875, -0.890625, -0.10540771484375, -0.53564453125, 0.27734375, 1.5517578125, 0.56787109375, -0.2357177734375, -0.2384033203125, -0.78076171875, -0.650390625, -0.31640625, 0.02764892578125, -1.0009765625, -1.1171875, 0.58642578125, -0.3095703125, 0.460693359375, -0.46337890625, -0.184326171875, -1.4033203125, -0.3388671875, 0.9658203125, -0.0758056640625, 0.197265625, -0.33740234375, -0.470703125, -1.2900390625, -0.130859375, 0.01355743408203125, -0.169189453125, 1.625, 1.4130859375, -0.78271484375, 1.2958984375, -1.0478515625, 2.158203125, -0.81005859375, 0.64306640625, 0.364990234375, 1.77734375, 0.3798828125, -0.52685546875, 0.77099609375, -2.029296875, -0.045074462890625, 0.423828125, -0.33203125, 0.04351806640625, 1.7470703125, 2.130859375, 0.7421875, 0.3466796875, 0.44189453125, 1.58203125, 1.4150390625, -0.460693359375, -1.365234375, 2.0, -0.87744140625, 0.75634765625, -0.111083984375, -0.2059326171875, -0.23193359375, 0.00868988037109375, -0.455810546875, 0.4326171875, -0.245849609375, 0.79541015625, 0.2275390625, 0.43896484375, 0.5224609375, 1.4482421875, 0.7373046875, 0.9287109375, 1.3798828125, -0.023956298828125, -0.76513671875, -0.08660888671875, 0.04083251953125, -0.06396484375, -0.7734375, -1.01171875, -0.9931640625, 0.90478515625, -0.85107421875, -1.1328125, 0.330810546875, 0.818359375, 0.299560546875, 0.361328125, 1.2861328125, -0.31396484375, 0.389892578125, -0.75732421875, 0.307861328125, 1.5595703125, -0.309326171875, -0.49169921875, 0.97021484375, -1.7216796875, 0.552734375, 0.459716796875, -1.3330078125, 0.4404296875, 1.1376953125, 0.37255859375, 0.272705078125, 0.39501953125, -0.25341796875, 1.8955078125, 0.79931640625, -0.30029296875, -0.8359375, 0.65478515625, 1.416015625, 1.3349609375, 0.6572265625, 2.12890625, 1.4736328125, 0.3916015625, 0.305419921875, 0.70556640625, 0.11932373046875, -0.904296875, -1.1123046875, -1.0830078125, -0.71533203125, -0.156494140625, -0.302734375, 1.58203125, -1.7294921875, 0.6533203125, -0.826171875, -1.68359375, 1.7314453125, 0.03680419921875, 0.1893310546875, -0.3095703125, 1.6142578125, 0.93115234375, -0.68798828125, 0.33740234375, -1.1240234375, -0.896484375, -0.5029296875, -0.323486328125, 0.11822509765625, 0.71484375, -1.0908203125, 1.1513671875, -0.54345703125, 0.220458984375, -1.3173828125, -1.22265625, -0.7275390625, -1.6494140625, 0.61767578125, 1.4921875, -1.5224609375, -0.802734375, 1.3583984375, -1.4453125, -1.7783203125, -0.291259765625, 1.826171875, 2.40234375, -0.76708984375, 0.4375, 0.2078857421875, -1.27734375, 0.06634521484375, -0.2451171875, 2.181640625, 0.84375, -0.560546875, 0.002349853515625, -1.3671875, 1.5087890625, -1.53515625, 0.85302734375, -0.65771484375, -0.85546875, 1.291015625, 0.63232421875, -0.63037109375, -0.466064453125, -0.2529296875, 2.291015625, 1.341796875, -0.29541015625, 0.296630859375, -0.412841796875, -0.37646484375, -1.47265625, -0.919921875, 2.25, 0.0064697265625, -0.64794921875, 0.3017578125, -0.450927734375, -2.3515625, -0.220703125, -1.0595703125, -0.76708984375, -1.58984375, -0.461181640625, -0.1920166015625, -0.1622314453125, 0.293212890625, 0.515625, -0.9189453125, -1.6201171875, -0.06439208984375, -0.35693359375, -0.97998046875, -0.1954345703125, 0.155029296875, 1.046875, 0.341064453125, -0.5458984375, 0.59423828125, -2.888671875, -1.5244140625, -0.479736328125, -0.802734375, -0.5458984375, -0.81982421875, -1.0244140625, 0.81298828125, 1.9091796875, 0.11981201171875, 0.0552978515625, -1.423828125, -0.5966796875, -0.60107421875, -1.220703125, 0.1900634765625, -0.1695556640625, -1.2587890625, 0.98486328125, -1.236328125, 0.444091796875, 0.75, 1.68359375, 0.28759765625, 0.339599609375, -0.994140625, -0.5166015625, 2.205078125, -0.6181640625, 0.3134765625, 1.22265625, -1.5078125, -0.177978515625, -1.271484375, 0.426513671875, -0.165283203125, -0.138671875, -1.568359375, -1.7294921875, -0.71337890625, 0.60546875, -0.0721435546875, -0.8251953125, 1.23828125, -0.04656982421875, 0.379150390625, -0.8544921875, -0.51318359375, 0.78173828125, -0.488525390625, 1.1767578125, 0.35888671875, 0.28125, -1.423828125, -0.98095703125, 2.1640625, 0.84619140625, 1.7158203125, -0.8955078125, 0.6455078125, 0.2421875, 1.076171875, -0.91162109375, -0.59423828125, -0.50537109375, 0.337158203125, -0.544921875, 1.7158203125, -0.0863037109375, -0.471923828125, -0.7041015625, 0.39794921875, -0.595703125, 1.5498046875, 0.84423828125, -1.5166015625, 2.14453125, 0.383056640625, -1.5361328125, 1.568359375, -1.185546875, 0.75439453125, -1.51953125, -0.6982421875, 0.45458984375, 1.0283203125, -1.6025390625, -0.30029296875, 1.0751953125, -0.70703125, 0.73974609375, -1.4248046875, -0.82568359375, -0.9658203125, -0.047088623046875, -0.259765625, 0.9013671875, 0.339111328125, 0.392333984375, 0.423095703125, 0.72509765625, -0.1751708984375, -0.60400390625, 0.0908203125, 1.5322265625, -1.3974609375, 0.638671875, 0.264892578125, 1.1337890625, 1.3134765625, 0.11676025390625, 0.258056640625, -0.85546875, 0.286865234375, -0.279296875, -0.4228515625, 0.1422119140625, -0.53515625, -0.87548828125, 0.82861328125, 0.68994140625, 0.3662109375, -0.1405029296875, -0.837890625, -1.0771484375, 0.63623046875, -0.72509765625, -0.492919921875, 0.19873046875, 0.76416015625, 1.5068359375, -0.66845703125, 0.64404296875, 0.270751953125, 0.246826171875, 0.331298828125, -0.3017578125, -0.389404296875, 0.64404296875, 0.350341796875, -0.51171875, 0.11444091796875, 2.140625, -0.08074951171875, -1.38671875, -0.460693359375, 0.349365234375, 0.521484375, -0.65966796875, 0.1038818359375, -0.0103912353515625, 0.1680908203125, -0.388427734375, 0.102294921875, 0.044097900390625, 0.183837890625, -0.36279296875, 0.444580078125, -1.099609375, 0.357666015625, 0.006610870361328125, -0.53466796875, -1.0810546875, -1.2119140625, 1.1630859375, 1.48046875, 0.2027587890625, -0.1737060546875, -0.392822265625, -1.390625, -0.16552734375, 0.71484375, -0.77001953125, 0.9794921875, -0.12200927734375, 1.4404296875, 0.59765625, -1.56640625, -0.94384765625, 0.0252685546875, -0.03997802734375, 0.78076171875, 0.428466796875, -0.2235107421875, 0.53857421875, 1.98828125, 0.56396484375, -0.62060546875, 0.395263671875, -0.374755859375, 0.0731201171875, 0.59765625, -0.390625, 0.5673828125, -0.533203125, -0.09307861328125, 1.978515625, 2.25390625, 0.12310791015625, 0.0804443359375, -0.37890625, -0.292236328125, -0.85400390625, -0.435302734375, -0.87841796875, -2.001953125, 1.16796875, 0.30322265625, -0.5078125, 0.23583984375, -0.08917236328125, -0.1455078125, 1.1865234375, 2.11328125, 0.115966796875, 1.298828125, -2.333984375, 0.81689453125, 3.369140625, 0.09783935546875, 0.6826171875, 0.3486328125, 1.0615234375, -1.7734375, -0.701171875, -0.483154296875, -0.282470703125, -1.74609375, -1.0771484375, 0.0430908203125, -0.66162109375, 1.732421875, -0.70947265625, -1.6298828125, 0.9853515625, 1.2041015625, -0.471923828125, 0.6865234375, -0.413818359375, 0.76513671875, 0.228271484375, 1.0341796875, -0.0345458984375, 0.7421875, 1.1728515625, -1.9150390625, -1.822265625, -1.07421875, -1.87109375, 2.28515625, 1.041015625, -0.35400390625, 1.4150390625, 1.703125, 0.4814453125, 0.5419921875, 0.71630859375, -0.98828125, 0.79541015625, -1.08984375, 0.05328369140625, -0.373779296875, -2.70703125, -2.291015625, 0.04388427734375, -0.97412109375, -0.53515625, 1.5166015625, 1.0654296875, -0.47021484375, -0.311279296875, 0.042510986328125, 1.3095703125, 0.249755859375, -0.7939453125, 1.3564453125, 0.927734375, 1.736328125, -1.21484375, 0.398193359375, -0.8603515625, -0.91064453125, 1.5791015625, -0.07537841796875, 2.150390625, 1.431640625, -0.68896484375, -0.58251953125, 0.132080078125, 0.8125, 0.483154296875, -0.64990234375, -0.07879638671875, 1.60546875, 0.9443359375, -1.802734375, -1.7353515625, -1.6630859375, -0.59912109375, 0.453125, -2.2734375, -0.1318359375, -0.87255859375, -1.388671875, 0.2119140625, -0.9111328125, -2.486328125, 0.1669921875, -0.80419921875, 0.89404296875, -1.033203125, -0.7392578125, -0.8046875, 2.640625, 2.119140625, 0.49169921875, -0.29150390625, 1.3974609375, 0.11456298828125, 0.95263671875, -0.070068359375, 0.287841796875, -1.1533203125, 0.03314208984375, 0.012969970703125, -0.72607421875, -0.1649169921875, -0.78662109375, -0.4873046875, 0.83251953125, 0.323486328125, 1.216796875, -0.99072265625, -0.7880859375, 0.5146484375, 0.061920166015625, -0.927734375, 1.0966796875, -1.052734375, -0.62890625, 0.68212890625, 1.314453125, -0.66943359375, -1.537109375, -0.32373046875, 0.114501953125, -0.136474609375, -0.5712890625, -0.1142578125, 0.474853515625, -2.103515625, 1.3193359375, 1.013671875, -0.2890625, 0.71728515625, -1.6201171875, -0.7822265625, 1.228515625, 1.1025390625, -0.79736328125, -0.2081298828125, -1.404296875, -0.67333984375, -0.117919921875, 0.06329345703125, -0.671875, 0.1634521484375, 2.15234375, 0.67578125, 0.4326171875, 1.1455078125, -0.267333984375, 0.75, 1.0625, 0.312744140625, -0.6669921875, 2.2265625, -0.8583984375, 1.021484375, 0.8525390625, 0.17822265625, -0.277099609375, -1.4638671875, -0.42822265625, -0.377197265625, -0.92919921875, 0.9033203125, 0.75732421875, -0.42529296875, 1.2666015625, 2.10546875, 1.2314453125, 1.0400390625, 2.220703125, 0.29736328125, -0.50244140625, -0.4423828125, -1.482421875, -1.193359375, 0.451171875, -0.178955078125, 1.248046875, -0.235595703125, -0.76025390625, -1.4345703125, -1.1767578125, 0.028839111328125, -1.9375, -0.76904296875, -1.76953125, -0.490478515625, -0.72314453125, 0.55908203125, 0.61181640625, 0.5009765625, 0.3291015625, -0.74609375, -1.1513671875, 1.9189453125, -1.2421875, -0.47314453125, 0.56494140625, 1.455078125, -0.4990234375, 0.211669921875, -0.1666259765625, -0.386474609375, -1.0302734375, 0.765625, -0.484375, 0.85986328125, -0.61083984375, -0.1629638671875, 0.59912109375, -1.5771484375, -1.345703125, 2.14453125, 0.5166015625, 1.0146484375, 0.48583984375, 0.900390625, 0.12066650390625, 1.677734375, -0.5390625, -0.14501953125, -2.498046875, -0.53466796875, 1.01171875, 1.3916015625, 0.71337890625, 1.6650390625, 0.458740234375, 0.67333984375, -0.50390625, -0.84716796875, 0.68603515625, -0.8837890625, 0.0982666015625, -0.53271484375, -0.04833984375, -1.2744140625, 0.50390625, 1.22265625, -0.05682373046875, -0.287109375, 0.243408203125, -0.2020263671875, 0.2176513671875, 0.900390625, -0.65673828125, -0.2279052734375, 0.92529296875, -0.10296630859375, -0.298828125, 0.057464599609375, 0.243896484375, 0.54541015625, 0.876953125, -1.134765625, -1.0625, 1.287109375, 0.62890625, -0.411865234375, -0.58154296875, -0.10919189453125, -0.61083984375, -1.5703125, 1.7314453125, 0.8623046875, -0.8935546875, 0.437255859375, -0.61767578125, 1.046875, 0.4814453125, 1.6357421875, 0.35595703125, -1.9306640625, 0.321533203125, 0.7109375, -0.728515625, 0.4619140625, -0.7490234375, -1.33984375, -0.95947265625, -0.0504150390625, 0.52783203125, 1.498046875, 0.08624267578125, 0.935546875, 0.390869140625, -0.58447265625, 1.2919921875, -0.64453125, -0.5947265625, -0.251953125, 0.515625, -1.0205078125, -1.931640625, -0.375, -1.076171875, -0.1529541015625, -0.310302734375, -1.318359375, 0.52685546875, 0.046875, 0.2119140625, 0.67236328125, -0.1314697265625, 0.66455078125, 2.958984375, 0.32470703125, 1.611328125, -0.265625, -0.1578369140625, -0.3466796875, -1.03125, -0.60205078125, 0.42431640625, 0.456787109375, -0.5791015625, -0.265625, 0.44189453125, -2.33984375, 0.07293701171875, 2.291015625, -0.1390380859375, 0.466064453125, 0.96240234375, 0.740234375, -1.37109375, -1.1376953125, -0.63134765625, -0.1798095703125, -0.96484375, 0.5693359375, -1.5966796875, -0.335205078125, 0.30859375, -1.6572265625, 0.28271484375, -1.625, 1.5146484375, -0.06707763671875, 1.9296875, 1.705078125, -0.51708984375, 0.85791015625, 1.294921875, -0.5517578125, 0.4482421875, -0.9873046875, 1.0869140625, -1.2353515625, 2.46484375, 0.304931640625, -0.19091796875, -0.244873046875, -0.77001953125, 0.66796875, 0.06201171875, 0.044403076171875, -1.5537109375, -0.0977783203125, -0.369384765625, 0.381103515625, 1.033203125, 1.09765625, -1.2421875, 0.09014892578125, 2.279296875, 0.37646484375, 0.60595703125, -0.2340087890625, -1.216796875, 2.2421875, -1.2900390625, 0.23779296875, -0.5546875, 0.264892578125, 0.79736328125, 1.728515625, 0.371826171875, -2.005859375, 0.91064453125, 0.1488037109375, -0.61328125, 1.9853515625, -0.75732421875, 0.435791015625, 0.845703125, -1.1044921875, 0.79833984375, 0.051513671875, -0.1014404296875, 0.662109375, 0.03582763671875, -3.048828125, 0.28466796875, -0.8662109375, 1.228515625, 1.5458984375, 0.43994140625, 0.7421875, 1.0107421875, 0.87744140625, -0.2379150390625, -0.41796875, 1.244140625, 1.052734375, -0.0787353515625, 1.271484375, 1.275390625, 1.4677734375, -0.445556640625, -0.2423095703125, -0.46484375, -0.10430908203125, 0.358642578125, -0.7158203125, -1.7421875, 0.42822265625, 0.22705078125, -1.1455078125, 0.2177734375, -1.177734375, -0.450927734375, 0.2900390625, 0.322998046875, -0.358154296875, 0.65283203125, -1.78125, -0.235595703125, -0.1571044921875, 0.916015625, -0.91455078125, 0.022979736328125, 0.54345703125, 1.623046875, -2.115234375, -0.268798828125, 1.2705078125, -0.4638671875, -0.884765625, 0.52392578125, -1.78515625, -1.2314453125, 1.41015625, -0.287353515625, 0.55859375, 2.353515625, -0.255859375, 2.025390625, -0.241455078125, -0.339599609375, 0.58349609375, -1.5419921875, -0.072509765625, -1.005859375, 0.382568359375, 0.81591796875, 0.2496337890625, 0.76416015625, -0.5361328125, 1.3037109375, -0.11328125, 1.0380859375, 0.3671875, 0.6650390625, 0.2196044921875, 0.6669921875, 0.5771484375, 1.3818359375, -0.0394287109375, 0.268798828125, 0.1627197265625, -0.06597900390625, 1.0673828125, -0.7294921875, -1.5791015625, -0.2783203125, -1.134765625, -0.043609619140625, -0.10565185546875, -1.1455078125, -1.169921875, -1.103515625, 0.499755859375, 2.123046875, 1.6357421875, 0.619140625, 0.447265625, 0.4228515625, -0.6845703125, 0.640625, -1.296875, 1.35546875, -1.1279296875, -0.5556640625, -0.31640625, 0.43017578125, 1.478515625, -0.032470703125, -0.6865234375, 0.09234619140625, -1.400390625, 1.31640625, 0.05596923828125, 0.9384765625, -0.56689453125, -1.4697265625, 0.054351806640625, 0.48095703125, 1.154296875, -0.00010758638381958008, 0.6171875, 1.2138671875, -0.43603515625, -1.2724609375, -0.60986328125, -1.599609375, -0.0152435302734375, 0.189208984375, 0.472900390625, -2.248046875, 0.187744140625, 0.873046875, -0.3310546875, 0.525390625, 1.78515625, -1.568359375, -0.34326171875, 0.7607421875, 0.4736328125, -0.267822265625, 0.75146484375, -2.373046875, 1.0986328125, 1.3916015625, -0.21435546875, 0.9541015625, 0.685546875, 0.4755859375, -0.79248046875, -0.93310546875, -0.4521484375, -3.203125, 0.6005859375, -1.5498046875, 0.19921875, -1.240234375, 1.130859375, 2.21484375, 0.7353515625, 0.443603515625, -0.386962890625, 0.37451171875, 0.44287109375, -0.2783203125, -0.299072265625, -0.541015625, -1.91015625, -1.4873046875, 0.9091796875, -2.318359375, 0.70751953125, -0.69482421875, 1.37109375, 0.6005859375, 0.384033203125, -0.99072265625, -1.015625, -1.783203125, 1.2099609375, 0.9091796875, 0.386962890625, -1.5439453125, -1.97265625, 1.5791015625, -0.11163330078125, -1.484375, 0.372802734375, -0.16259765625, -0.58544921875, -0.6982421875, -0.16796875, -1.26953125, -1.5849609375, -1.4541015625, -0.0859375, 1.078125, 0.062164306640625, 0.268798828125, -0.92626953125, 1.2822265625, -2.087890625, 0.56591796875, 1.4619140625, -0.73486328125, 0.35693359375, -0.2305908203125, -0.0782470703125, 0.52197265625, 0.238525390625, 0.270751953125, 0.44482421875, 0.5927734375, -1.3134765625, -1.6962890625, -0.63427734375, -1.087890625, 0.8955078125, -0.82275390625, -0.5791015625, -0.37451171875, 0.9267578125, -0.046173095703125, -2.009765625, -0.2032470703125, 0.30224609375, -0.90087890625, 0.458984375, -0.46923828125, 0.389892578125, 0.018035888671875, 0.80322265625, 0.0933837890625, -0.402587890625, -0.2130126953125, 0.1275634765625, -1.46875, 0.69677734375, -1.3154296875, -0.08111572265625, -0.1265869140625, -0.5361328125, 0.64111328125, 1.857421875, -0.994140625, 1.0244140625, 0.324462890625, 0.53466796875, -1.0390625, -0.440673828125, 0.377685546875, -1.8828125, 1.736328125, 0.1888427734375, 0.0758056640625, -0.350830078125, -0.8525390625, -1.55859375, -0.75244140625, -1.255859375, -0.33447265625, -0.04437255859375, -0.1729736328125, -1.0166015625, 0.1036376953125, -0.73974609375, -1.83203125, -1.5986328125, 1.2314453125, -0.482421875, -0.4501953125, -0.1944580078125, -0.39697265625, 1.7783203125, -0.025665283203125, 0.9814453125, 1.4833984375, 0.57763671875, 0.732421875, 1.1220703125, 2.357421875, 0.28369140625, 0.042938232421875, -0.4072265625, -1.453125, 0.31884765625, -0.7978515625, 0.90478515625, 0.1065673828125, 1.2900390625, -0.46240234375, -0.396240234375, 0.098388671875, -1.041015625, 0.1884765625, -0.5126953125, 0.189697265625, 1.060546875, 1.2587890625, 0.55419921875, 2.21875, -0.40966796875, -0.7353515625, 1.630859375, 0.01311492919921875, -0.08160400390625, 1.3828125, 0.90185546875, 2.056640625, -1.5078125, -3.138671875, 1.501953125, -0.939453125, 0.70458984375, -1.5166015625, 0.1688232421875, 1.1142578125, 1.203125, 0.439208984375, 0.1239013671875, -0.07794189453125, 0.032501220703125, -0.63916015625, 0.0078582763671875, -1.3818359375, -0.890625, 0.1212158203125, -0.12548828125, -0.1925048828125, 1.8720703125, -0.46435546875, 0.95556640625, 1.2646484375, 0.055755615234375, 0.387939453125, -0.33935546875, 0.86767578125, -0.05560302734375, -1.23828125, 0.11614990234375, -0.285400390625, -0.2313232421875, 0.7060546875, -0.363525390625, -0.09393310546875, -0.2313232421875, -0.72412109375, 1.4609375, 1.115234375, 1.021484375, -0.6884765625, 0.92138671875, -0.36572265625, 0.0836181640625, 2.4140625, 0.6484375, 0.54931640625, 0.47802734375, 0.4072265625, 1.5537109375, -1.0263671875, -2.05078125, 0.35205078125, 1.1201171875, -1.2880859375, 0.237548828125, 0.381103515625, -1.396484375, 0.79931640625, -0.53125, -1.3427734375, 0.61865234375, 0.357666015625, 1.3095703125, 0.30126953125, 1.2841796875, 0.2147216796875, -0.0293426513671875, 1.10546875, 1.6328125, 0.1424560546875, -2.001953125, 1.9853515625, -1.4267578125, -1.28125, 0.00170135498046875, 0.24951171875, 0.47314453125, -0.9228515625, -1.9287109375, -0.92578125, 1.0234375, -0.8525390625, 0.6943359375, -0.475830078125, 0.3798828125, -0.60986328125, -0.1865234375, -0.58056640625, 1.1005859375, -1.1689453125, -1.09765625, -0.8125, -1.892578125, -0.623046875, 0.619140625, 0.4599609375, 1.943359375, 0.245361328125, -0.10455322265625, 0.97412109375, -1.171875, -0.2191162109375, -0.2098388671875, 0.91357421875, -0.4248046875, -0.474365234375, -0.53173828125, -0.7939453125, 0.456298828125, 0.3134765625, 0.5048828125, -0.1717529296875, -0.80615234375, -0.299560546875, -0.56103515625, 0.224853515625, 1.09765625, 0.343994140625, -0.5859375, -0.309814453125, 2.359375, 0.25830078125, 0.720703125, -0.11370849609375, -0.2296142578125, -0.463623046875, -0.71630859375, 0.2413330078125, 0.5625, 0.33935546875, -0.345703125, 1.572265625, 0.42578125, -0.94921875, 0.5712890625, -2.27734375, -0.53125, -0.09197998046875, -1.416015625, 0.09130859375, -0.36279296875, -2.0390625, -0.262939453125, -0.44677734375, -0.03741455078125, -0.60009765625, 1.6708984375, 0.78662109375, -1.5361328125, 1.736328125, -0.487060546875, 1.421875, -0.173583984375, -1.3857421875, 1.0771484375, -0.595703125, 0.77099609375, -0.9208984375, -1.0869140625, 1.525390625, 0.2298583984375, -0.002655029296875, 0.876953125, -0.72265625, 1.4892578125, -1.7490234375, 0.61865234375, -0.8056640625, 0.08453369140625, -0.63427734375, -1.0751953125, -2.11328125, 0.3359375, 1.0361328125, 1.1494140625, -1.1787109375, -0.82080078125, 1.056640625, -0.0164947509765625, 2.443359375, -1.478515625, 0.178955078125, 1.6259765625, -1.3095703125, 0.71240234375, 1.1416015625, -0.2724609375, -1.82421875, -0.6162109375, 0.56103515625, 0.103515625, -0.65869140625, 0.83935546875, -0.1614990234375, -0.56640625, -0.890625, 0.441650390625, 1.619140625, -1.5205078125, -2.00390625, -0.5517578125, -0.400634765625, -1.591796875, 0.24462890625, -1.70703125, 0.0697021484375, 2.1953125, -1.6298828125, 1.3466796875, -1.908203125, 0.07958984375, 0.384765625, -2.384765625, 1.1064453125, 0.011199951171875, -0.6083984375, -1.6064453125, 1.115234375, 0.3603515625, 1.8779296875, -1.126953125, -0.62255859375, 0.11968994140625, -0.68603515625, -1.0703125, 0.1741943359375, -0.10205078125, 0.97265625, -1.53125, -1.302734375, -1.8681640625, -0.31591796875, 0.494873046875, -1.107421875, 0.53662109375, 0.203857421875, 0.212158203125, 0.02008056640625, -1.0625, 0.416748046875, 1.6240234375, 0.37841796875, -0.2457275390625, -0.98046875, 1.25390625, -1.2529296875, 0.7138671875, 0.39013671875, -0.01546478271484375, 0.55126953125, -0.261474609375, 0.69091796875, -0.896484375, -0.0033206939697265625, 0.052978515625, -1.4365234375, 1.1982421875, -0.86474609375, -0.060638427734375, -0.556640625, -0.728515625, -1.91796875, -0.478759765625, -1.0361328125, 0.8818359375, -0.82275390625, -0.183837890625, 1.142578125, 0.5849609375, 0.480224609375, 0.49755859375, 0.61669921875, -0.2132568359375, -2.982421875, 1.490234375, 0.357421875, 2.099609375, -0.28955078125, 0.72900390625, -1.08984375, -0.05523681640625, -0.403564453125, -0.395751953125, -0.521484375, -0.57421875, 0.3193359375, 0.181640625, -0.0219879150390625, 0.81884765625, 0.05950927734375, -1.248046875, 0.951171875, -0.5732421875, 0.0154876708984375, 0.447265625, -0.255126953125, 0.3837890625, 0.56396484375, -1.6669921875, 0.460693359375, -0.31591796875, 0.08184814453125, 1.4169921875, 0.310302734375, 1.44921875, 1.4287109375, 0.888671875, -0.042388916015625, -0.78173828125, 0.41650390625, -1.7587890625, -0.82861328125, 1.21484375, 0.351318359375, 2.052734375, -0.2998046875, 0.42578125, 1.2490234375, -1.15234375, -0.2030029296875, -0.48095703125, 0.002735137939453125, -2.001953125, -0.298095703125, 0.32421875, 0.315673828125, -1.1787109375, -1.0888671875, -0.183349609375, 1.6083984375, -1.5712890625, 1.9189453125, 0.85302734375, 0.83447265625, -1.25, -1.1982421875, 1.7470703125, 2.04296875, -1.54296875, 0.44580078125, -0.253662109375, -0.2978515625, -0.7763671875, -0.56591796875, 0.11376953125, -0.7919921875, 1.0234375, 0.3193359375, 0.161376953125, 1.7919921875, 0.14404296875, -0.77490234375, 0.78076171875, 0.349609375, -0.97802734375, -0.9658203125, -0.62353515625, -0.68359375, -1.5419921875, -1.3876953125, -0.369384765625, 0.70703125, -0.423095703125, 0.218017578125, -1.07421875, -0.865234375, -0.5166015625, 0.08489990234375, -1.1015625, -1.556640625, -0.5341796875, 0.8046875, -0.9404296875, 0.298828125, -0.9765625, -0.2044677734375, -2.666015625, 0.237060546875, 1.1953125, -0.6953125, -1.3994140625, -0.209228515625, 0.2763671875, -0.333251953125, -0.8095703125, 0.003154754638671875, -0.94287109375, 0.6484375, -0.50146484375, -1.357421875, -0.049468994140625, 0.10943603515625, 1.935546875, 0.07647705078125, -0.78369140625, -0.5078125, -1.77734375, -0.75537109375, 1.7568359375, -0.1370849609375, 0.71875, 0.1541748046875, 0.43212890625, -0.451171875, -1.1728515625, 0.0677490234375, 1.6162109375, -2.087890625, 0.86083984375, -1.744140625, -0.341796875, -0.57470703125, 1.01171875, -1.5498046875, 0.54345703125, -1.37109375, 0.79833984375, -0.80859375, 0.904296875, -0.40234375, -0.96826171875, -0.367919921875, 0.203125, 0.73583984375, 1.125, 1.056640625, 0.265869140625, 1.30859375, -0.12548828125, -1.857421875, -2.58203125, -0.2744140625, 0.06280517578125, -0.751953125, 0.60986328125, 1.4638671875, -1.3916015625, -0.78662109375, 0.5595703125, -0.07086181640625, -0.662109375, 0.14013671875, 0.22021484375, 0.87451171875, 0.048492431640625, 2.03515625, -0.3193359375, -0.54443359375, 0.302001953125, -0.2919921875, -0.9345703125, 0.23193359375, 0.56884765625, -0.7626953125, -0.60693359375, 1.041015625, -0.411376953125, -0.006267547607421875, -0.75537109375, 0.06707763671875, 0.435546875, 1.4765625, 1.337890625, -0.7294921875, -0.08087158203125, 0.90771484375, -1.875, -0.32373046875, 1.75, -0.99755859375, -1.021484375, -0.580078125, -0.56201171875, 1.2421875, -0.69677734375, -0.6396484375, 0.1954345703125, 1.43359375, 0.38525390625, 0.236083984375, -1.4970703125, -0.51123046875, -0.18212890625, 0.66748046875, -1.1982421875, -0.406982421875, -0.249267578125, 0.53173828125, -1.6416015625, -0.1072998046875, 0.205322265625, -0.79248046875, 0.298095703125, 1.2353515625, 1.7900390625, 1.158203125, -0.01093292236328125, -2.177734375, -0.11199951171875, 0.0173797607421875, 0.354736328125, -2.02734375, 0.34716796875, -0.275146484375, -0.67919921875, -0.75634765625, -0.289306640625, -1.16015625, -0.986328125, -0.5322265625, 0.260009765625, 0.30078125, 0.6962890625, -1.490234375, 0.78369140625, 1.1923828125, 0.82421875, 0.30419921875, -0.8388671875, -0.454833984375, -0.9580078125, -0.42138671875, -0.1375732421875, -0.8798828125, -0.77294921875, 0.60546875, 1.8388671875, -0.58349609375, 1.291015625, -0.0546875, 0.252685546875, 0.76513671875, 0.2685546875, 0.1693115234375, -1.2607421875, 1.845703125, -0.75, 0.373779296875, -2.23828125, -1.279296875, -0.54833984375, -0.68896484375, -0.0487060546875, -0.603515625, -1.2470703125, -0.412109375, 1.0556640625, 0.6318359375, 1.99609375, 0.67138671875, -0.89306640625, -0.384765625, 2.330078125, 1.96484375, 1.6494140625, -0.85107421875, -0.325927734375, 2.041015625, -0.2467041015625, -1.6513671875, 2.259765625, 0.07080078125, 1.3271484375, -0.7431640625, -0.783203125, 1.2763671875, -0.7470703125, 1.9970703125, 0.9287109375, 2.59375, 0.5498046875, -0.305908203125, 0.297607421875, 2.021484375, -1.2978515625, 1.26171875, 0.406005859375, -0.5263671875, 0.927734375, -1.091796875, -0.76611328125, -1.314453125, 1.24609375, -0.08209228515625, -0.822265625, 0.1654052734375, 1.552734375, -0.28759765625, -0.30615234375, -0.52587890625, 0.2607421875, 0.48583984375, -0.2469482421875, 2.052734375, 2.3359375, 0.489013671875, 0.0477294921875, 0.2042236328125, 1.37109375, 0.363525390625, -0.88134765625, -0.70703125, -0.78125, 0.37158203125, 2.01171875, -1.447265625, 0.028778076171875, -0.95849609375, 0.6328125, 0.133544921875, -0.70654296875, 0.9765625, -1.34375, 0.4814453125, 0.7080078125, 0.33154296875, 0.46875, -0.1812744140625, -0.87158203125, -0.045684814453125, -1.0234375, -0.7021484375, -0.21142578125, -0.8310546875, 0.298095703125, 1.1904296875, -0.923828125, 1.486328125, 0.4169921875, 1.8642578125, -1.1337890625, 2.30859375, -0.55517578125, -0.990234375, 0.1683349609375, -1.3740234375, 1.4404296875, -0.466064453125, 1.357421875, 0.059051513671875, 1.7041015625, -1.4326171875, 0.34619140625, 0.490234375, 0.08551025390625, 1.787109375, 0.9560546875, 0.92333984375, 1.0712890625, 0.8037109375, 0.39111328125, -2.3125, -0.37841796875, -0.7021484375, 2.0625, -0.5009765625, 0.68798828125, 0.0787353515625, 0.355712890625, -0.421630859375, -1.0244140625, -0.03668212890625, 1.7412109375, -1.748046875, 0.65283203125, 1.4296875, -0.416748046875, 1.9775390625, 1.5302734375, 1.2685546875, 0.195556640625, -1.583984375, -0.284912109375, -0.01824951171875, -0.00704193115234375, 0.09442138671875, 1.3203125, -1.384765625, 0.138916015625, -0.021697998046875, -0.476318359375, -1.39453125, -0.08612060546875, -0.33251953125, -1.8212890625, 0.7275390625, -1.1708984375, 1.0458984375, -0.1719970703125, -0.68115234375, 1.9169921875, 0.5986328125, -0.314453125, 0.31494140625, 1.7998046875, -1.6318359375, 2.646484375, 0.45068359375, 0.09637451171875, 1.1162109375, -0.748046875, 0.4697265625, -0.9228515625, -1.1943359375, -0.66162109375, -0.50830078125, 0.68310546875, -1.2685546875, -0.2288818359375, 0.61328125, -0.734375, 0.56982421875, -0.6201171875, 0.34619140625, 0.487060546875, 1.876953125, 0.80224609375, 1.783203125, -0.6171875, -0.277099609375, -0.90185546875, -1.9873046875, 0.142578125, -0.83203125, 0.21826171875, 0.59619140625, -1.037109375, -1.3740234375, -2.052734375, 0.434326171875, -0.642578125, 0.11370849609375, -0.030914306640625, 1.119140625, 0.256103515625, 1.5927734375, 0.44580078125, -0.52001953125, -0.94091796875, 0.38720703125, 0.568359375, -2.044921875, -0.10687255859375, -1.3056640625, -0.051361083984375, 0.79052734375, 2.0390625, 0.92529296875, 0.3662109375, -0.6923828125, 0.174560546875, 0.9248046875, 0.1583251953125, 0.499267578125, 0.29150390625, -0.962890625, 0.140380859375, -0.06494140625, -1.142578125, 0.034271240234375, -0.74169921875, -0.6259765625, -0.0823974609375, -0.07830810546875, -0.638671875, 0.451416015625, 0.59912109375, -1.01953125, 1.19140625, 0.994140625, 1.94921875, 1.09765625, 0.5107421875, -0.257080078125, 0.857421875, -1.0546875, -0.6494140625, 1.490234375, -0.267578125, 0.7724609375, -0.314453125, -1.7041015625, 1.15234375, 1.1484375, -0.1875, 0.041595458984375, 0.495849609375, 1.73046875, -0.6171875, -0.99267578125, 0.52392578125, 0.1292724609375, -0.60400390625, 1.2626953125, -1.2099609375, -0.403564453125, -2.560546875, -2.138671875, -0.84423828125, 0.09320068359375, -1.1748046875, -1.1904296875, -1.05859375, -1.140625, 0.6474609375, -1.232421875, -0.499267578125, -0.2303466796875, -0.98193359375, 0.75146484375, -0.31005859375, 0.4267578125, -1.6259765625, 0.1416015625, 0.900390625, -0.0936279296875, -0.70703125, -0.78369140625, -1.0380859375, -0.416748046875, 0.6689453125, -0.59228515625, -1.1064453125, 1.00390625, 0.10504150390625, -0.6142578125, -1.0888671875, 1.220703125, -1.3330078125, 0.3798828125, -1.396484375, -1.0478515625, -0.099365234375, 0.022125244140625, -0.047393798828125, 0.0599365234375, -0.35205078125, -0.8359375, -1.689453125, 0.479248046875, 1.4404296875, 2.068359375, -2.431640625, 1.2724609375, 0.03729248046875, -0.293212890625, -0.08660888671875, -0.69921875, -0.07073974609375, 0.8193359375, 0.43017578125, -0.089599609375, 0.5302734375, -0.8876953125, 0.94384765625, 0.671875, 0.01861572265625, 0.376708984375, -1.80859375, 0.53564453125, -0.689453125, -1.146484375, 0.0380859375, -1.15625, -1.3154296875, 0.55078125, 0.51513671875, 0.1561279296875, -1.2578125, 1.5234375, -0.20849609375, 1.0205078125, 1.2109375, -0.03131103515625, -0.8447265625, -1.3671875, -1.2529296875, 0.08831787109375, -0.65478515625, 0.172119140625, 0.0046844482421875, 0.85400390625, -1.6826171875, -1.4033203125, 1.7255859375, -0.92822265625, 0.578125, 0.109130859375, 0.2744140625, 0.513671875, -0.716796875, -0.2347412109375, -0.08154296875, 1.6484375, -0.51416015625, 0.6962890625, -0.14013671875, 1.00390625, 1.48828125, 0.0227508544921875, 0.83740234375, 0.7197265625, 0.3056640625, 0.4560546875, -0.0950927734375, -1.5625, -0.52001953125, -0.66796875, -1.1337890625, 0.06878662109375, 1.1474609375, -1.044921875, -0.8095703125, -0.005519866943359375, -0.045318603515625, -1.6220703125, 0.78759765625, -0.279296875, -0.12939453125, -0.305419921875, -0.11370849609375, 1.970703125, 0.4443359375, -0.2283935546875, -1.5244140625, -0.2344970703125, 0.0831298828125, -1.541015625, -0.490966796875, 0.1282958984375, -0.85205078125, -0.04693603515625, 0.1839599609375, -0.97998046875, 1.1728515625, 1.0087890625, 0.783203125, -2.521484375, -1.125, -0.90673828125, 1.2919921875, -0.93359375, -1.28125, 0.490478515625, 1.28515625, -1.072265625, -1.6728515625, 1.09375, -0.978515625, 0.489013671875, 0.11431884765625, -0.74755859375, 0.9111328125, -0.82421875, 1.5166015625, -0.9453125, -1.255859375, -0.254150390625, 0.99365234375, -0.0006399154663085938, 1.55078125, -1.638671875, -0.351318359375, -0.1776123046875, -0.88525390625, 0.37744140625, 0.21337890625, 0.80712890625, -1.4638671875, -0.31298828125, -0.449951171875, 1.1650390625, 0.0816650390625, 0.8466796875, -1.017578125, 0.08990478515625, 0.2425537109375, -0.37646484375, -0.990234375, 1.306640625, -0.313720703125, 2.4296875, 1.95703125, -0.234375, 1.1318359375, -0.320556640625, 1.2880859375, 1.150390625, 0.0271453857421875, -1.109375, -0.2381591796875, 0.2303466796875, -0.5625, -0.323486328125, 2.568359375, 0.9794921875, -0.5263671875, -0.69921875, -1.6005859375, -0.7822265625, -0.96142578125, -0.88134765625, 1.3330078125, -1.2099609375, -0.166015625, 0.1104736328125, 1.587890625, 0.724609375, -1.419921875, -2.259765625, 0.168212890625, -1.158203125, -1.01953125, 0.418701171875, 0.427734375, 0.44287109375, 1.5087890625, 0.88427734375, 0.21044921875, 0.414306640625, -1.431640625, 1.5625, -0.9970703125, 0.46484375, 0.260986328125, -0.7724609375, 1.7431640625, -0.48486328125, 0.62060546875, -0.027099609375, -1.3828125, 0.7978515625, 0.2423095703125, 0.83251953125, -0.2335205078125, -0.456298828125, 0.061920166015625, -0.0380859375, -1.7470703125, 0.400146484375, -1.720703125, 0.004749298095703125, -0.818359375, -1.1923828125, -1.220703125, 2.66796875, -0.4794921875, 0.059326171875, -0.4130859375, -1.3349609375, -0.73681640625, 0.75, 2.130859375, -0.56591796875, -1.11328125, 1.91796875, 1.58203125, 0.6455078125, -0.154541015625, -0.466064453125, -0.341796875, 0.50927734375, -0.0692138671875, 0.05718994140625, -0.457763671875, -0.837890625, 1.1552734375, -1.5107421875, 0.46875, 0.86865234375, 0.40576171875, 2.123046875, -0.0178070068359375, -1.5283203125, -0.81396484375, 0.6162109375, 1.552734375, 0.5517578125, 0.08843994140625, -1.3427734375, -0.10931396484375, -1.259765625, -0.068115234375, 1.169921875, 1.6708984375, 0.447998046875, 0.578125, 1.0302734375, 0.89990234375, 0.2410888671875, -0.451904296875, 0.489013671875, -1.0478515625, 0.372314453125, 0.10546875, -1.5654296875, 0.318359375, -0.7021484375, -0.065185546875, -0.10589599609375, -0.1632080078125, 0.82275390625, -1.0966796875, 0.497802734375, -1.57421875, -0.29443359375, -1.0927734375, -0.251708984375, 0.86572265625, 1.16015625, 0.82958984375, -2.12109375, 0.1551513671875, -0.44873046875, -0.0714111328125, -0.28857421875, -0.26953125, -0.6181640625, -0.07080078125, -0.91796875, 1.505859375, 0.32177734375, 0.783203125, 0.54638671875, -1.78515625, 0.72314453125, 0.55615234375, 0.72607421875, -0.1793212890625, -0.0174407958984375, 1.9970703125, -1.9111328125, 0.36572265625, 0.253662109375, 1.9970703125, 0.0931396484375, -1.119140625, -0.261962890625, -0.6943359375, 0.70654296875, 0.68017578125, -0.697265625, -0.1412353515625, 0.5146484375, -0.14697265625, 0.371826171875, 0.6728515625, -0.50244140625, 0.1593017578125, 1.3720703125, -0.005451202392578125, -0.3935546875, -0.95947265625, 0.08447265625, -0.947265625, 0.01515960693359375, 0.1988525390625, 1.41015625, -0.77783203125, 0.55029296875, 0.95263671875, -0.047882080078125, 0.1849365234375, -0.97265625, 2.099609375, -0.298583984375, 1.2275390625, -0.1387939453125, 1.2119140625, 1.15625, 0.226318359375, 0.92431640625, 0.16552734375, -1.978515625, 0.587890625, 1.2939453125, 1.4404296875, -0.35009765625, -0.71435546875, 0.50146484375, -0.15576171875, -0.833984375, 0.307373046875, 0.41552734375, -0.5380859375, -0.521484375, 1.40625, -0.36083984375, 1.39453125, 1.4052734375, -1.482421875, -0.8759765625, 0.71875, -1.4111328125, 0.0391845703125, -0.365966796875, -0.38720703125, -0.315673828125, 0.330810546875, 1.029296875, -0.3193359375, -0.362060546875, -0.73486328125, -0.30517578125, 0.060272216796875, 0.23388671875, -1.056640625, -0.2044677734375, 0.67822265625, 0.283935546875, 0.1405029296875, 0.6728515625, 0.1778564453125, 1.1015625, 1.0341796875, -1.3056640625, 0.2344970703125, 0.1724853515625, -1.2783203125, -2.1484375, -0.85498046875, -0.64013671875, 1.9326171875, -1.5, -1.8154296875, -0.91748046875, -2.001953125, -0.7158203125, -0.7177734375, -0.60009765625, 0.96630859375, -0.0911865234375, -0.1483154296875, -0.91943359375, -1.5439453125, 0.57470703125, 0.72021484375, -0.0655517578125, 0.1602783203125, 1.7431640625, 0.1702880859375, 1.4755859375, -0.439208984375, 0.1065673828125, -1.6484375, -0.65576171875, 0.02569580078125, -0.1923828125, 0.2244873046875, 0.66064453125, 0.2457275390625, 1.5498046875, 0.056793212890625, -2.0078125, -0.167236328125, 0.192138671875, -0.26953125, -1.392578125, 0.56005859375, -0.5615234375, -0.58837890625, 0.6513671875, 0.388427734375, 2.28515625, 2.318359375, 0.0330810546875, -0.5302734375, -1.2314453125, 1.6279296875, -0.72265625, 0.67626953125, -0.266845703125, 0.36865234375, -0.078857421875, -0.2479248046875, -0.3623046875, -1.9599609375, 1.2978515625, -0.7734375, -0.7607421875, 0.68505859375, 0.057403564453125, -1.1806640625, 1.0966796875, 0.110595703125, -0.428955078125, 2.546875, -0.49267578125, 0.050933837890625, -0.54541015625, -1.134765625, 1.0888671875, -1.3408203125, -1.1943359375, -1.0234375, 1.28125, 0.392822265625, 0.0657958984375, -2.470703125, 0.51171875, -0.84912109375, 0.71630859375, 1.3681640625, -0.15576171875, 1.1865234375, 1.671875, -0.254638671875, -1.3193359375, -1.46484375, 0.6826171875, -0.65087890625, -0.07330322265625, -0.1700439453125, -0.09478759765625, -0.248779296875, 0.50390625, -0.80224609375, -1.671875, 1.09765625, 1.083984375, 0.44873046875, 1.1875, 0.896484375, -0.21044921875, -0.0205230712890625, 0.1773681640625, -0.93603515625, 0.247802734375, 0.487060546875, -1.0859375, 1.208984375, 0.884765625, -0.1685791015625, -0.26025390625, -0.278076171875, -0.7216796875, 0.35986328125, -2.2578125, -0.6865234375, -0.66357421875, 1.0751953125, -0.0111541748046875, 0.06878662109375, -0.1536865234375, 0.42822265625, -0.1976318359375, -1.7158203125, -1.310546875, 0.54638671875, 1.005859375, -0.42822265625, -0.4345703125, -0.3125, 0.072509765625, 0.166015625, 1.4296875, -0.53271484375, -0.91455078125, 1.615234375, -0.392578125, -1.072265625, -0.52587890625, 1.82421875, 2.390625, -0.29345703125, 0.51123046875, -0.1953125, 1.041015625, -0.1361083984375, -1.17578125, -0.35302734375, -0.1790771484375, -0.25341796875, -0.7265625, 0.410400390625, 0.0980224609375, -0.7158203125, -1.66015625, 0.2099609375, -1.556640625, -1.787109375, -1.072265625, -0.56689453125, -0.1417236328125, 0.473388671875, -1.36328125, -0.08160400390625, -0.4736328125, -0.068603515625, 0.323486328125, 1.634765625, -0.05230712890625, -0.1842041015625, 0.66357421875, 1.970703125, -0.888671875, -0.56494140625, -0.1788330078125, 2.490234375, 0.638671875, 0.7958984375, 1.75390625, 0.65576171875, 0.8857421875, 0.25, -0.407470703125, -3.1015625, 0.7900390625, -1.755859375, -0.71826171875, 0.412841796875, 1.6171875, 2.2734375, -0.8271484375, -1.294921875, 0.10955810546875, -1.6376953125, -0.927734375, 0.2529296875, -0.2308349609375, -0.72802734375, 0.034759521484375, -0.333984375, 0.333740234375, 1.38671875, -1.22265625, -0.96533203125, -0.55712890625, -0.68359375, 1.0302734375, 1.6767578125, -0.1282958984375, 0.2432861328125, 1.5654296875, 1.3955078125, 1.025390625, 0.85009765625, 0.66552734375, -0.464599609375, 2.0234375, 0.7236328125, -1.912109375, 0.178466796875, -1.0888671875, -0.56201171875, -1.775390625, -0.720703125, -1.123046875, 0.0379638671875, -0.65185546875, 1.9931640625, -0.08856201171875, -0.392578125, -1.0205078125, -1.8388671875, -0.07867431640625, -0.46630859375, 0.388671875, 0.697265625, 0.63037109375, -1.1015625, -0.6953125, 0.2318115234375, 0.73046875, -0.25830078125, -0.83154296875, -0.1453857421875, 1.0205078125, 1.06640625, 0.035186767578125, -0.99609375, -0.041778564453125, -0.33740234375, -1.0341796875, -0.274658203125, 0.1259765625, -0.6357421875, -0.759765625, 0.006710052490234375, -1.521484375, -0.7392578125, 0.56005859375, -0.18701171875, -0.091796875, -0.2529296875, 1.759765625, 2.361328125, -0.19677734375, 0.0174407958984375, -0.85546875, 0.6806640625, -0.10089111328125, -0.607421875, 0.37060546875, -0.419921875, 0.23486328125, 0.415283203125, 0.30322265625, 0.00890350341796875, -0.79833984375, -0.05615234375, -2.21875, -0.71337890625, -0.032745361328125, -1.2314453125, -0.6376953125, -0.2076416015625, 0.751953125, -0.4521484375, -0.2144775390625, -2.673828125, 0.1375732421875, 0.2083740234375, -0.65673828125, -0.412841796875, 1.697265625, -0.317138671875, -0.08843994140625, -0.51318359375, 1.1982421875, -0.74755859375, -0.4755859375, 1.197265625, -0.04705810546875, 0.2392578125, 0.52734375, 0.86865234375, 2.251953125, -0.337890625, 0.7509765625, 2.05859375, 1.59375, 1.1728515625, 0.2269287109375, 0.1224365234375, 1.6845703125, -0.329833984375, -1.2158203125, 1.0859375, 1.55859375, 0.00799560546875, 0.1744384765625, 2.828125, -0.80126953125, -1.6015625, 0.1671142578125, 0.2479248046875, -0.282958984375, -1.009765625, 0.1614990234375, -0.11663818359375, -0.62646484375, -0.9853515625, -0.0296783447265625, 1.1826171875, -1.8994140625, 1.3779296875, 0.7626953125, -0.61376953125, 0.97119140625, -0.413818359375, -2.22265625, -0.88427734375, 0.457763671875, 0.0074615478515625, -1.1015625, 0.03802490234375, 1.8642578125, 1.58203125, -0.73974609375, -0.11358642578125, 0.24169921875, -0.2193603515625, 1.2080078125, 0.62939453125, -2.291015625, 2.109375, -0.2076416015625, -1.5087890625, -1.0029296875, 0.356201171875, 0.60546875, -0.1318359375, 0.83203125, -0.04083251953125, 1.6982421875, 1.021484375, -1.8623046875, -0.6533203125, 0.78662109375, -0.2308349609375, -0.367919921875, -1.0439453125, -0.87109375, 0.2001953125, 0.476318359375, -0.38818359375, -0.241455078125, -0.7119140625, 0.14404296875, -0.07061767578125, -0.68603515625, -0.398681640625, -0.4951171875, 2.408203125, 0.560546875, -0.7744140625, -0.1427001953125, 0.796875, 0.76806640625, -1.6591796875, -0.365478515625, 1.244140625, 1.28515625, 1.3984375, -0.31884765625, -0.0753173828125, -0.33984375, -0.0289459228515625, 0.63427734375, 1.85546875, 1.90234375, 1.5849609375, 0.4794921875, 0.78466796875, 1.2060546875, -0.6220703125, 0.171875, 0.2264404296875, 0.221435546875, -1.294921875, 1.8447265625, -0.86669921875, -0.91162109375, -0.35400390625, -0.1422119140625, 1.28125, 1.1484375, -1.341796875, 1.15625, 1.251953125, 0.478515625, 0.1221923828125, 0.9765625, 0.8251953125, 0.0184173583984375, 0.65185546875, 1.154296875, 0.1954345703125, 1.2265625, 0.264892578125, 0.654296875, -0.6259765625, 1.9033203125, -0.11553955078125, -0.49951171875, -0.27197265625, 1.8076171875, 1.8037109375, 0.5615234375, -1.59765625, 1.4228515625, -0.396484375, 1.056640625, 0.0247802734375, -1.2158203125, 0.345703125, -0.482421875, -0.1566162109375, 0.361328125, -0.75048828125, 0.57177734375, -1.29296875, 1.1826171875, -0.73095703125, -1.52734375, -0.49267578125, 1.1923828125, -0.426513671875, 0.13330078125, -0.466064453125, -0.0570068359375, 1.2685546875, -0.87451171875, -1.5712890625, 0.1732177734375, 0.1409912109375, -1.58984375, -0.3740234375, 0.480224609375, 1.029296875, 0.84033203125, -2.689453125, 0.8603515625, -0.947265625, 0.90478515625, -0.28515625, -1.5693359375, 3.21484375, 1.095703125, 0.4736328125, -0.59521484375, -0.06842041015625, 0.77783203125, 0.98388671875, -0.68505859375, 0.77294921875, -0.480224609375, 0.09051513671875, -1.09765625, -0.059967041015625, -1.1611328125, -1.2138671875, -0.2081298828125, -0.0142059326171875, 0.7734375, -0.271484375, 0.54541015625, 0.33935546875, 0.52783203125, 0.53759765625, 0.66796875, 0.1021728515625, -1.1240234375, -1.974609375, 0.1500244140625, 0.307373046875, 0.95263671875, 1.896484375, 0.318359375, 1.3203125, -1.548828125, -1.333984375, 0.52197265625, 0.307373046875, 1.5009765625, 1.736328125, 1.1298828125, -0.767578125, -0.0306549072265625, 1.79296875, 1.2041015625, 1.298828125, -0.133544921875, -0.88720703125, -1.5625, -1.60546875, 0.2430419921875, -0.01166534423828125, 0.85888671875, -0.2489013671875, -1.7265625, -0.1470947265625, -0.0270233154296875, -1.923828125, -0.72265625, -1.029296875, -0.82861328125, 1.4189453125, 0.0261688232421875, -1.6552734375, 0.09417724609375, 1.794921875, -0.1658935546875, 1.5205078125, -2.2890625, -1.0859375, -1.060546875, 0.298828125, 0.437255859375, -0.326904296875, 2.0234375, 0.1920166015625, -0.73388671875, -0.681640625, 1.220703125, 1.01953125, 0.595703125, 0.310546875, -0.50244140625, -0.038787841796875, 0.145263671875, 0.0780029296875, -2.08984375, 0.00994110107421875, -0.46875, -0.46142578125, -1.578125, -0.19677734375, -0.283935546875, 0.064208984375, 0.472900390625, 1.43359375, -0.87451171875, -0.5400390625, -0.93994140625, 0.6953125, -0.07122802734375, -0.5810546875, 0.236083984375, 0.50732421875, 0.69140625, -0.203857421875, -0.8837890625, 1.30859375, -0.2457275390625, -0.1375732421875, 2.546875, -1.1650390625, -1.5859375, -2.599609375, 0.1192626953125, -0.5087890625, 1.6982421875, 0.270263671875, -0.3310546875, -0.131103515625, -0.5810546875, -1.1884765625, -0.006351470947265625, -0.294189453125, -0.515625, -0.2255859375, -0.281982421875, 0.04736328125, -0.13330078125, 3.341796875, 0.66650390625, 1.341796875, 1.7177734375, 0.0261077880859375, 1.0556640625, 0.232666015625, 1.15625, 0.90771484375, 1.1591796875, 0.556640625, -0.3798828125, 0.428955078125, 2.05078125, -0.28955078125, -0.57861328125, -0.87841796875, 0.1845703125, -0.328857421875, 0.32958984375, 0.76025390625, -0.982421875, -0.30517578125, 1.9150390625, 0.94287109375, -0.1275634765625, -0.430908203125, 1.0673828125, 0.06781005859375, -1.154296875, -0.189208984375, -0.58935546875, -0.07550048828125, 0.5830078125, -0.021636962890625, 1.0751953125, 1.375, 1.2958984375, -0.52783203125, -0.4541015625, -0.375244140625, 0.9677734375, 0.232666015625, -0.11309814453125, -0.51171875, -2.330078125, 1.0048828125, -1.48828125, 1.39453125, -0.14111328125, -0.86962890625, 0.08349609375, -1.2802734375, 1.359375, 0.450439453125, 1.208984375, 0.75390625, 0.93603515625, -0.57861328125, -0.268310546875, -1.17578125, -0.331787109375, 0.18701171875, 0.1307373046875, -0.12548828125, 0.5869140625, 0.180908203125, -1.6064453125, -0.75927734375, -1.00390625, -0.06805419921875, 0.2100830078125, -1.0849609375, -1.74609375, 0.03570556640625, 0.11566162109375, -0.04705810546875, 0.00545501708984375, 0.58154296875, 0.6767578125, 1.2392578125, -0.927734375, 0.10345458984375, -1.83984375, 1.048828125, -0.552734375, -0.056396484375, 2.009765625, -1.52734375, -0.90576171875, -0.380126953125, -1.1748046875, -1.0166015625, -3.412109375, 0.56787109375, -0.005390167236328125, 0.2442626953125, 0.60693359375, -0.045684814453125, 0.501953125, -0.79443359375, -1.541015625, 2.056640625, -0.103759765625, -0.2005615234375, 1.373046875, -0.3935546875, 0.051788330078125, -0.313232421875, 0.1712646484375, 0.2315673828125, 0.48583984375, -0.7841796875, 0.6103515625, -0.2724609375, -0.58935546875, 1.513671875, -0.55859375, 1.4853515625, 0.53369140625, 0.44970703125, 0.95361328125, -0.578125, -0.25537109375, 0.258544921875, 0.438720703125, -0.69873046875, 0.0506591796875, -1.9833984375, -0.927734375, -0.09375, 1.298828125, 0.76318359375, 2.3125, -0.32958984375, -0.7021484375, -1.435546875, -0.307373046875, 0.5224609375, 0.41015625, 0.56884765625, 0.09063720703125, 0.10797119140625, -0.1544189453125, 1.5400390625, 1.2412109375, 0.75927734375, 0.79638671875, 1.7958984375, -0.0014657974243164062, -0.61572265625, 1.291015625, -0.18701171875, -0.75146484375, -0.09527587890625, -1.900390625, 0.1744384765625, 0.8388671875, -1.388671875, 0.5029296875, 1.4287109375, -0.81494140625, 0.9501953125, 0.271728515625, 0.6572265625, 1.1103515625, 1.654296875, -3.107421875, 0.1494140625, 0.298095703125, 0.032257080078125, 0.4375, 2.021484375, 1.7734375, 0.65478515625, 0.3984375, 0.611328125, 0.490478515625, -1.103515625, 1.0517578125, 1.0859375, -0.032470703125, -1.322265625, 0.74365234375, 0.1387939453125, -0.509765625, -1.041015625, -1.58984375, -0.89404296875, -1.0625, 1.31640625, 0.08685302734375, 0.67138671875, -0.380615234375, -0.073974609375, 1.4873046875, -0.76416015625, -0.99853515625, -0.158203125, 0.464111328125, -0.78662109375, 0.026885986328125, -2.560546875, -0.0999755859375, -0.479736328125, -0.318115234375, 1.650390625, 0.08544921875, -0.473876953125, 0.52978515625, 1.5869140625, -1.71875, 1.3095703125, -2.705078125, -0.352783203125, 1.751953125, -0.55224609375, 0.50732421875, 1.4609375, 0.8505859375, -0.64306640625, -0.7919921875, 0.374755859375, -0.373779296875, -1.3486328125, -0.46435546875, -1.26171875, -0.390869140625, 0.607421875, 1.052734375, 0.061614990234375, 2.30078125, 0.1910400390625, -0.865234375, 1.263671875, -0.2064208984375, 1.2314453125, 0.2281494140625, -0.06939697265625, -1.2109375, -0.200439453125, 1.1455078125, -0.955078125, -0.744140625, 0.60400390625, 0.9375, 0.6796875, -0.043243408203125, 0.8408203125, -0.90234375, 0.7607421875, 1.17578125, -0.21484375, 1.1171875, -0.6337890625, 0.3837890625, -0.58154296875, 0.62158203125, 0.0278167724609375, -2.306640625, 2.271484375, -0.2427978515625, 1.0791015625, -1.107421875, 0.5908203125, -0.79638671875, -0.2490234375, -1.658203125, -0.55517578125, 0.56005859375, 1.4580078125, -1.6943359375, 0.30908203125, -0.350341796875, 0.568359375, 1.4306640625, -0.198974609375, -1.99609375, -0.095947265625, -0.65673828125, 0.1121826171875, 0.60205078125, -1.439453125, -0.5224609375, -0.441162109375, -1.080078125, -1.01953125, -0.90234375, 0.13671875, -0.66796875, 0.0599365234375, -0.44921875, 0.68017578125, 0.20654296875, -0.45703125, -0.5361328125, 0.36474609375, 0.2427978515625, -0.85009765625, -0.8232421875, -0.41943359375, 0.19775390625, -2.443359375, -0.85986328125, -0.1419677734375, -0.93701171875, 0.1943359375, 0.424560546875, 0.1876220703125, -0.1337890625, 0.28857421875, 0.86376953125, -0.393310546875, 0.74365234375, 0.1015625, -1.1748046875, 0.3759765625, 0.85693359375, -1.4326171875, 0.9716796875, -0.033416748046875, 0.59423828125, 0.450927734375, -0.84033203125, -2.3984375, 1.2626953125, -0.9013671875, 0.154052734375, -1.36328125, 1.09375, -1.921875, 0.66015625, 0.4873046875, 0.274658203125, 0.2978515625, 0.5595703125, -0.32080078125, -0.471923828125, 1.4521484375, 0.3623046875, -0.9814453125, -1.4599609375, 0.64892578125, 0.72607421875, 0.09246826171875, 1.095703125, -0.50439453125, 0.0182342529296875, 0.71533203125, 1.37109375, 0.16650390625, -0.459716796875, 0.451416015625, -1.02734375, 0.2252197265625, -0.062103271484375, -0.6328125, 0.822265625, -0.3896484375, 0.64599609375, -0.32421875, 1.2197265625, -0.338623046875, -0.87646484375, 0.281005859375, 0.69677734375, 0.3515625, -0.11572265625, -0.7451171875, 0.9892578125, 0.267578125, 0.771484375, 1.2138671875, 1.859375, 0.2154541015625, 0.476318359375, -1.1103515625, -1.728515625, 0.68994140625, -0.164306640625, 0.54150390625, -1.078125, -0.434814453125, -0.98291015625, -1.1025390625, 0.0269927978515625, -1.197265625, 0.65576171875, -0.385498046875, 0.62255859375, 0.4716796875, -0.94384765625, -1.7353515625, 0.1905517578125, 0.489013671875, 1.50390625, -0.0850830078125, 0.884765625, 0.277099609375, -0.13330078125, -0.039764404296875, 0.423095703125, -0.053619384765625, -0.2275390625, -0.17529296875, -1.8408203125, -0.21728515625, -0.5810546875, -0.5576171875, -1.255859375, -1.1181640625, 0.58447265625, 0.9287109375, 0.58544921875, 0.96142578125, -0.25390625, 0.87255859375, -1.990234375, 1.73046875, 0.62939453125, 0.5419921875, -1.18359375, 0.53369140625, -0.310546875, -0.359130859375, 0.11505126953125, 1.263671875, -1.6572265625, 0.00093841552734375, -0.3662109375, -0.93359375, -0.64599609375, 0.8515625, 0.2447509765625, -1.099609375, -1.18359375, 0.209716796875, -0.20361328125, 0.334228515625, 1.56640625, 0.71142578125, 0.0024356842041015625, -0.560546875, 0.165771484375, -0.0035552978515625, 0.8134765625, 0.787109375, -1.818359375, 0.47314453125, 0.7646484375, 0.476806640625, -0.2166748046875, -0.595703125, -0.437255859375, 0.99462890625, -1.2314453125, -0.30126953125, -0.76904296875, -0.59228515625, -0.7705078125, -0.09765625, -1.60546875, -0.222900390625, -0.6494140625, -0.268310546875, 1.626953125, 0.2017822265625, -1.806640625, 0.44384765625, -0.72216796875, -0.68017578125, -0.7158203125, 1.5224609375, 1.560546875, 2.361328125, -0.90771484375, -0.318115234375, 0.27001953125, 0.42431640625, 0.87744140625, 0.385498046875, 0.003963470458984375, -0.73681640625, 0.7080078125, -0.486083984375, 0.09600830078125, -1.2763671875, 0.94091796875, -2.087890625, 0.07501220703125, 1.9541015625, -0.00801849365234375, -2.0859375, 0.716796875, -1.7822265625, 0.07855224609375, -0.05194091796875, -0.061431884765625, 0.76025390625, 0.6298828125, -1.7490234375, -1.9501953125, 1.45703125, 0.1541748046875, 0.032379150390625, -0.64306640625, 0.1939697265625, 1.115234375, 0.45703125, 0.51953125, 1.2841796875, 1.55859375, 0.4501953125, 0.25927734375, 1.021484375, -0.5458984375, 0.41455078125, 0.72119140625, 0.1610107421875, 0.11981201171875, 0.03985595703125, 0.93505859375, 0.5849609375, -0.26904296875, 0.41943359375, 0.69921875, -0.50732421875, 1.388671875, -0.2423095703125, -0.283447265625, 1.162109375, -0.317138671875, 0.8974609375, 0.5126953125, -0.64892578125, 1.9697265625, -1.1611328125, 0.3828125, 0.66650390625, 2.0390625, -0.2301025390625, -0.60107421875, 0.2088623046875, -0.73828125, -0.443115234375, -0.04425048828125, 2.505859375, -0.02239990234375, -0.369873046875, 0.327392578125, -0.225830078125, 0.1038818359375, -0.71337890625, -0.341796875, 1.689453125, -0.018829345703125, 0.11883544921875, 0.5947265625, 0.343994140625, -1.3359375, 1.0234375, 0.59765625, -1.0224609375, 1.1005859375, 1.658203125, 0.329833984375, -0.0728759765625, -0.055633544921875, 0.8779296875, -2.32421875, 1.2451171875, 1.7021484375, 1.361328125, -0.515625, 1.5888671875, 1.822265625, 0.95703125, -1.158203125, 0.66162109375, 0.6806640625, 0.62255859375, -0.83203125, 1.5576171875, 1.1728515625, 0.2056884765625, -1.056640625, 0.339111328125, -0.60205078125, 0.065185546875, -1.5625, -1.5556640625, -0.33837890625, -0.81689453125, 1.1005859375, -1.173828125, 0.6552734375, -0.85302734375, 1.0908203125, -1.0498046875, 1.05078125, 0.468505859375, 0.98193359375, -1.150390625, -0.58544921875, 1.28125, 0.26708984375, -0.7421875, -0.6787109375, 1.123046875, -1.578125, -1.640625, 0.869140625, 0.329833984375, 0.2427978515625, 0.053802490234375, 0.55029296875, -0.437255859375, 1.470703125, -0.5263671875, -1.330078125, -0.68115234375, -0.853515625, 0.2169189453125, 1.18359375, -0.88232421875, 0.40478515625, 0.022705078125, -0.39111328125, 1.41015625, -0.326416015625, 1.328125, 0.494873046875, -0.1629638671875, -1.1103515625, -0.80712890625, 0.33154296875, 0.390380859375, 0.450927734375, -0.85888671875, -1.1015625, 0.11834716796875, 0.57861328125, 0.297607421875, -1.0498046875, 0.55224609375, 1.0859375, -0.79931640625, -0.58349609375, 0.7314453125, 1.0341796875, 1.5595703125, 1.2666015625, -0.9716796875, -0.425048828125, 0.81982421875, -0.41259765625, -1.841796875, 2.318359375, 0.5390625, 1.8486328125, 1.1279296875, -0.56884765625, -0.323486328125, -1.0341796875, 0.2359619140625, -0.7158203125, 0.95263671875, -0.5361328125, 1.259765625, -0.273193359375, 0.09930419921875, -1.7373046875, 2.033203125, 1.4443359375, 0.4326171875, -0.728515625, -1.1513671875, 1.201171875, 0.02447509765625, 1.8447265625, -0.1434326171875, 0.01061248779296875, 0.484619140625, 1.55859375, 0.595703125, 0.93505859375, 0.72802734375, 0.434326171875, 2.091796875, -1.8583984375, -0.285888671875, -1.5, 1.6767578125, 1.3076171875, -0.465576171875, -1.3837890625, 1.91015625, 1.353515625, 0.90771484375, 1.1005859375, 0.1280517578125, 0.30029296875, -1.2607421875, 1.177734375, -0.11712646484375, 0.1090087890625, -1.1025390625, -0.181396484375, 0.360107421875, 1.126953125, 0.720703125, -0.00919342041015625, 0.473876953125, 1.091796875, 0.42822265625, -0.296142578125, 0.34716796875, -0.008453369140625, -2.365234375, 1.0888671875, -0.94384765625, 1.236328125, 1.44140625, 0.483154296875, -0.11566162109375, -0.498291015625, -0.381103515625, 1.376953125, -1.0224609375, -0.196044921875, -0.1273193359375, 0.2398681640625, -0.5859375, 0.451171875, 0.061248779296875, -1.4658203125, -2.365234375, 1.318359375, 0.32080078125, 2.076171875, -0.82958984375, -0.0885009765625, 1.453125, 0.6259765625, 0.292724609375, -1.287109375, 0.70166015625, -0.284912109375, -0.285888671875, -1.0126953125, -0.1553955078125, -0.4423828125, 0.02447509765625, 1.7490234375, -1.50390625, -0.007381439208984375, -1.462890625, -0.9814453125, 0.1279296875, 0.63818359375, 0.09906005859375, 0.2548828125, 0.351318359375, 0.481201171875, 0.09381103515625, 0.41943359375, 0.209716796875, 0.55419921875, -1.63671875, 1.2451171875, 0.99560546875, 0.94677734375, -1.224609375, 1.3134765625, -0.46875, -1.1337890625, -0.03350830078125, -0.327392578125, 1.779296875, -1.5576171875, 0.337158203125, -0.9453125, -1.0732421875, -0.6689453125, -0.057159423828125, -0.5908203125, 0.51806640625, -0.1998291015625, 0.06744384765625, -0.476318359375, -2.853515625, -0.69921875, -2.142578125, -1.0537109375, -0.423583984375, -0.26025390625, 1.4716796875, 1.5078125, 1.046875, -2.109375, -0.434326171875, 0.86865234375, -0.0963134765625, 1.58984375, 0.958984375, 0.859375, 0.25732421875, -0.192138671875, 0.258056640625, -1.9228515625, 0.947265625, -0.40771484375, 1.201171875, 0.56591796875, 1.232421875, -0.2568359375, 0.88232421875, -0.609375, -0.70556640625, -0.39892578125, -1.2431640625, 0.79052734375, -0.6611328125, -1.216796875, 0.611328125, -0.32275390625, -0.81982421875, -0.281494140625, 1.9765625, -3.306640625, 0.39111328125, 0.90625, -0.46728515625, -1.716796875, -0.8525390625, 0.501953125, -1.9892578125, 1.05859375, -0.611328125, 0.6767578125, 1.115234375, 0.371826171875, 1.6474609375, -0.67236328125, 0.485595703125, 1.8427734375, 0.260498046875, 0.60986328125, 0.97119140625, 1.09765625, 0.9853515625, 0.50146484375, 0.8330078125, -1.548828125, -0.129150390625, 0.033050537109375, 0.214599609375, -0.87353515625, 0.309326171875, 1.8330078125, 0.1416015625, 0.54833984375, -0.6328125, 0.284423828125, 0.0158843994140625, 0.4482421875, 0.58642578125, -0.0263214111328125, -0.400390625, -1.0166015625, 0.258544921875, -1.2685546875, -2.37890625, -0.57958984375, 0.442138671875, 0.2188720703125, -1.09375, -0.33154296875, 0.533203125, 0.5810546875, -1.2685546875, -1.8505859375, 1.4658203125, -0.4521484375, -1.1875, -0.35107421875, 0.7041015625, -0.490966796875, 0.98193359375, -1.04296875, 1.255859375, -1.2734375, 0.0987548828125, -0.389892578125, 0.1070556640625, 0.25244140625, -1.0751953125, 1.3056640625, -0.1259765625, 0.00579833984375, 0.306640625, -0.52734375, -0.8505859375, 1.0927734375, 0.08868408203125, 0.85400390625, 0.234130859375, 0.814453125, -0.177490234375, 0.339111328125, -0.469482421875, 0.5673828125, 0.483642578125, 0.1607666015625, -0.9091796875, -0.35498046875, -1.0185546875, 2.46484375, 0.6591796875, -1.28125, -0.57373046875, 0.8134765625, 0.8291015625, -0.3154296875, 0.469482421875, 0.09735107421875, -0.50048828125, 0.322998046875, -1.4404296875, 0.41455078125, 0.71337890625, 0.128173828125, -0.0733642578125, 0.57177734375, 1.103515625, -1.50390625, -0.4853515625, 0.425048828125, -0.0927734375, -0.7861328125, 0.201904296875, 0.2119140625, 0.91845703125, -0.0411376953125, -0.32666015625, -0.108154296875, 1.513671875, 1.3369140625, -1.4677734375, 0.2388916015625, 1.2197265625, 0.60546875, 0.99658203125, 0.74072265625, 2.337890625, -1.5703125, -0.6181640625, 0.81396484375, -0.405517578125, -0.90869140625, -0.37255859375, 0.65283203125, -0.252685546875, 0.45166015625, -2.474609375, -0.2470703125, 0.7001953125, 2.833984375, 0.7177734375, -0.2034912109375, -0.986328125, 0.3369140625, -0.9521484375, 0.60791015625, -1.0205078125, -1.330078125, -0.51904296875, 0.2437744140625, -1.16015625, 1.890625, -1.041015625, -0.93310546875, 0.97802734375, -0.463623046875, -0.87939453125, 0.67919921875, -1.919921875, -0.04705810546875, -0.1024169921875, -0.4794921875, -0.314453125, 0.281494140625, 2.23828125, -1.7275390625, -0.20458984375, 1.19140625, -1.4140625, 0.2042236328125, 2.20703125, -0.130859375, 0.837890625, -1.9833984375, -0.5166015625, 0.8427734375, -1.2158203125, 1.8994140625, 0.6904296875, -0.228271484375, -0.5029296875, -2.447265625, -0.036865234375, 0.83837890625, 1.5244140625, 1.369140625, -0.0268402099609375, -0.73193359375, -0.9375, 0.68212890625, -1.3603515625, -0.56591796875, 1.1328125, 0.0657958984375, 1.7099609375, -0.341064453125, 0.62744140625, 0.5244140625, -2.328125, 1.6787109375, -0.7900390625, -1.1162109375, -0.6044921875, -1.7236328125, 0.287353515625, -1.96484375, 1.537109375, 1.396484375, -0.156494140625, -0.1282958984375, -1.0380859375, 0.8359375, -0.8994140625, 0.587890625, 1.0830078125, 0.1015625, 0.6845703125, 0.55908203125, -0.6064453125, -0.3515625, 1.830078125, -2.037109375, -0.51806640625, -1.5, -2.0859375, 0.6708984375, -0.327392578125, 1.08203125, -1.794921875, -0.5556640625, 0.64404296875, 0.42041015625, -0.161376953125, 0.2081298828125, 0.32763671875, -1.6435546875, 0.72607421875, -1.4990234375, -0.36669921875, 0.7080078125, 0.23095703125, -1.4228515625, 0.6064453125, -0.406494140625, 0.337158203125, -0.9970703125, -1.498046875, 0.2056884765625, -0.1661376953125, -0.9716796875, -0.64453125, -0.90673828125, 0.241943359375, -0.490478515625, 2.53125, 1.2958984375, -0.33203125, 1.3974609375, 1.66796875, -0.50048828125, 0.09857177734375, -0.375, -0.024566650390625, 1.359375, -0.396484375, -0.654296875, 0.284912109375, 1.0400390625, -2.369140625, -0.54248046875, 1.6455078125, 2.03125, -0.451416015625, 0.1700439453125, -0.1527099609375, 1.65234375, 2.244140625, -1.0263671875, 0.28515625, 0.2978515625, 0.96337890625, 0.415283203125, -1.2041015625, 0.91064453125, 0.513671875, 0.383056640625, -0.9736328125, -0.3134765625, 1.615234375, -1.708984375, -0.10455322265625, 0.498779296875, -0.38037109375, -1.1845703125, -1.5087890625, 0.046356201171875, 0.354248046875, -1.14453125, -0.4248046875, 0.53662109375, 1.3125, 0.37060546875, 0.54833984375, -0.196044921875, -1.427734375, -0.317138671875, 0.1409912109375, -0.104736328125, 2.9375, 0.0369873046875, -0.29541015625, -1.7568359375, 1.0859375, 0.76708984375, -0.5615234375, 1.2509765625, -0.791015625, 0.95556640625, -0.407958984375, 1.97265625, 0.59521484375, 0.24365234375, 0.2266845703125, 0.60205078125, -0.136962890625, -0.52392578125, -1.0849609375, 0.77783203125, 0.76318359375, -0.32470703125, 0.681640625, 1.8115234375, -1.2255859375, -0.80419921875, -1.5205078125, -0.07733154296875, 0.5908203125, 0.09686279296875, -0.39697265625, 0.6181640625, 1.0849609375, -0.10296630859375, -0.36181640625, -0.171630859375, 0.3251953125, -1.484375, 2.923828125, 0.59521484375, -0.5078125, -2.078125, 0.266357421875, 0.1541748046875, -0.76806640625, -1.6552734375, 0.3935546875, -0.349365234375, -0.075439453125, 0.362060546875, 0.93701171875, -0.0054779052734375, -0.10107421875, -1.92578125, 0.74267578125, -0.2310791015625, -1.2080078125, -0.272216796875, -2.4453125, 0.84521484375, 1.0751953125, 0.0885009765625, -2.15234375, -0.092041015625, -0.57177734375, 1.13671875, 1.451171875, 0.129638671875, -1.80859375, -2.099609375, -0.0806884765625, 2.28515625, 1.923828125, 0.376220703125, -1.1669921875, 2.11328125, 0.0107421875, -0.55859375, 0.0251922607421875, -1.5634765625, 0.908203125, 0.280029296875, 1.20703125, -1.0341796875, -0.57421875, 1.84375, -0.89453125, -0.724609375, 0.38525390625, -1.3466796875, 0.308349609375, 0.2078857421875, 0.428955078125, -0.78125, -1.1689453125, 0.382080078125, 0.1197509765625, -0.1751708984375, 0.97607421875, -0.97314453125, 1.029296875, 0.72021484375, -0.578125, -0.55908203125, 0.2509765625, -0.681640625, 1.3515625, 0.1888427734375, 0.1854248046875, -0.70263671875, 1.1728515625, 0.262451171875, -0.460205078125, 1.0439453125, -2.708984375, 1.3037109375, 0.8818359375, -0.2042236328125, -0.576171875, 0.7607421875, -0.71826171875, 0.1075439453125, -0.48193359375, 0.572265625, -0.461181640625, 0.1268310546875, 2.369140625, 0.6279296875, -0.7119140625, 2.64453125, -0.53369140625, -0.33740234375, 0.8447265625, -0.5009765625, 0.289306640625, -0.2001953125, 1.61328125, 0.09637451171875, 1.0859375, -0.02252197265625, -1.25390625, 1.9482421875, -0.88720703125, 0.266845703125, 2.8828125, -0.78564453125, 1.69140625, 1.138671875, 0.9111328125, -0.0899658203125, -1.056640625, -1.111328125, -0.71875, -1.4541015625, 0.78173828125, 0.0295257568359375, 0.0557861328125, -0.4169921875, 0.8115234375, -0.3369140625, 0.97607421875, 1.3427734375, 0.4443359375, 0.26220703125, 0.85888671875, 0.1614990234375, 1.1474609375, -0.6708984375, -2.271484375, 1.271484375, -0.300048828125, 0.346923828125, -0.33056640625, -0.578125, -0.9365234375, 0.380615234375, 0.619140625, -2.72265625, 1.47265625, 1.1767578125, 0.76806640625, 0.59423828125, 0.1170654296875, -0.84765625, -0.603515625, -0.89794921875, -0.5009765625, -0.30224609375, -0.372314453125, -1.7890625, 0.91357421875, -2.15234375, 0.73193359375, -0.5546875, 0.6142578125, 0.35107421875, -0.57568359375, 0.1082763671875, -1.162109375, 0.12939453125, 0.07025146484375, 0.9814453125, -1.04296875, 1.8056640625, 0.703125, -1.802734375, 1.794921875, 0.018280029296875, -0.017364501953125, -0.06884765625, -0.86279296875, 0.7392578125, 0.6416015625, -1.3818359375, -0.76171875, -1.1796875, 0.72265625, 1.5703125, 0.92626953125, 1.521484375, -0.1588134765625, -0.6796875, -0.73583984375, -1.28125, 0.9833984375, -0.80615234375, -0.333251953125, 0.771484375, -0.70556640625, 0.4560546875, -0.8701171875, 0.7890625, 0.962890625, 0.205810546875, 0.473388671875, -0.47509765625, 0.01186370849609375, -0.88525390625, 0.465576171875, -0.703125, -0.0211639404296875, 0.34228515625, -2.244140625, 0.86279296875, 0.34521484375, -0.8212890625, 0.347900390625, 0.426025390625, -0.63330078125, -0.53662109375, -0.787109375, -1.2939453125, -0.00830841064453125, 1.58203125, -0.307861328125, -0.2479248046875, -2.919921875, -0.63623046875, -1.4833984375, -0.359375, 3.46484375, 0.5205078125, -0.697265625, -0.29443359375, -0.348876953125, -0.380615234375, -0.75634765625, 0.50732421875, 1.1943359375, -0.93505859375, 1.0234375, -2.234375, -1.494140625, 0.5107421875, 1.015625, 0.031402587890625, -0.427734375, 0.73828125, 0.78662109375, 0.72607421875, -0.29443359375, -0.421875, 1.080078125, -1.5302734375, 0.5009765625, -0.057342529296875, 0.6015625, 0.07000732421875, 0.2288818359375, -0.63525390625, -0.85400390625, -0.08917236328125, 1.4814453125, -0.79345703125, -0.8212890625, -0.10540771484375, 0.1793212890625, 0.36572265625, -0.0806884765625, 0.91064453125, -0.270263671875, -0.30322265625, 0.75048828125, -0.287109375, 0.09796142578125, 0.6181640625, -0.8740234375, -1.5146484375, -0.763671875, -0.046112060546875, 0.395263671875, 1.3994140625, -0.74169921875, 1.0791015625, -1.138671875, -0.92041015625, 0.452880859375, -1.861328125, 0.73046875, -0.63232421875, -2.021484375, -1.390625, 0.07110595703125, -0.8017578125, -1.791015625, -0.48291015625, 1.125, 0.3466796875, -0.85888671875, 0.318115234375, -0.036224365234375, -1.6650390625, 0.56884765625, -0.08380126953125, 0.12158203125, 1.3525390625, 1.0517578125, -0.779296875, -0.80126953125, -0.57958984375, 0.8447265625, 0.55859375, 0.1121826171875, -1.572265625, -0.666015625, -0.193359375, 0.85107421875, 1.80078125, 0.05255126953125, 2.0859375, -0.0589599609375, -1.0263671875, -1.0400390625, 0.8271484375, -0.96044921875, 0.57763671875, 1.3583984375, -0.08575439453125, -0.73974609375, -0.93798828125, -0.41015625, 0.59228515625, 1.7509765625, -0.40087890625, 2.58203125, 0.35791015625, -0.369384765625, 1.3515625, -0.85302734375, 0.368896484375, -0.24169921875, -0.8603515625, 0.685546875, -0.68798828125, -0.7587890625, -0.77294921875, 1.3154296875, 0.415283203125, 0.3173828125, 0.08935546875, 0.4443359375, -0.71142578125, -0.501953125, 0.90380859375, 0.91796875, 0.15625, 1.8671875, -0.76611328125, -0.9873046875, -0.12548828125, 1.5791015625, -0.1656494140625, -1.18359375, -0.65283203125, 0.081298828125, -3.037109375, 0.43115234375, -1.9501953125, 1.822265625, -0.48291015625, -0.4501953125, -0.3759765625, -0.005313873291015625, 0.1719970703125, 0.5732421875, 1.4521484375, 0.63134765625, -0.184814453125, 1.6640625, -0.328369140625, 1.34375, -1.388671875, -0.384765625, -0.685546875, 0.062103271484375, 0.0279998779296875, -0.0908203125, -0.580078125, -0.5361328125, -0.92822265625, -0.1083984375, -0.4853515625, -1.4443359375, 1.298828125, -0.62451171875, -0.87841796875, 1.267578125, 0.9248046875, -0.0941162109375, -1.05859375, 2.55078125, -2.275390625, 1.7578125, -0.73046875, -1.044921875, 0.16845703125, -0.8369140625, 0.87939453125, -0.44189453125, -0.67724609375, -0.9228515625, 1.9833984375, -0.0714111328125, 0.5224609375, -0.233154296875, -1.306640625, 1.4541015625, 1.7216796875, 0.9580078125, 1.36328125, -0.356689453125, -0.19287109375, 0.70849609375, 1.162109375, -0.2352294921875, -1.59765625, -0.97021484375, -0.0305938720703125, -0.798828125, 0.69189453125, 0.06268310546875, 0.99951171875, 0.312744140625, 1.1240234375, 0.01091766357421875, -1.9814453125, 0.1876220703125, -0.5498046875, 0.69091796875, -0.7060546875, 2.24609375, -0.2364501953125, -1.283203125, -0.654296875, -0.57421875, 0.006259918212890625, -0.81884765625, -0.017059326171875, 0.59228515625, -1.078125, -1.8193359375, -1.3974609375, 0.344482421875, 0.70703125, -1.775390625, 0.89111328125, 0.029998779296875, 0.5556640625, -0.1392822265625, 0.47705078125, 0.343994140625, -1.2275390625, -0.006481170654296875, -0.88623046875, 1.3583984375, 0.66259765625, -0.53076171875, -1.3212890625, 0.2041015625, -1.0205078125, -1.3232421875, -0.798828125, 1.6044921875, -0.3095703125, 2.314453125, -0.455322265625, 0.28955078125, -1.8701171875, 0.9072265625, -0.79638671875, 0.96240234375, 0.7255859375, 0.79541015625, -0.53125, -0.52001953125, 1.251953125, 1.66015625, 0.7783203125, -0.0887451171875, 1.3203125, -0.293212890625, -0.314208984375, -1.533203125, -0.1605224609375, 0.68798828125, 1.2724609375, 0.7041015625, 0.5947265625, 0.79345703125, -0.442138671875, -0.73681640625, -0.2183837890625, 0.414306640625, 2.39453125, 2.189453125, 0.60400390625, -0.49658203125, 0.346923828125, 0.56005859375, -0.8095703125, 1.326171875, 1.1591796875, -0.763671875, 0.158203125, -1.0927734375, 1.5791015625, 0.80126953125, 0.638671875, 0.467529296875, -0.7421875, 1.0966796875, -0.7509765625, -1.6953125, -0.7490234375, -0.78857421875, -1.5751953125, -0.7314453125, -1.103515625, -0.386474609375, -1.90234375, -1.2197265625, 0.7724609375, -0.274658203125, 0.53466796875, 0.5556640625, 1.6728515625, 0.9716796875, -1.9482421875, -0.1497802734375, -0.6748046875, -0.0286102294921875, 1.5703125, 0.84130859375, 0.235107421875, -0.77685546875, 0.0226593017578125, -0.92041015625, -0.294921875, 0.55615234375, -0.8203125, 0.453369140625, -1.9560546875, 0.01274871826171875, -1.9814453125, 0.82861328125, -0.80224609375, 0.10888671875, -0.1046142578125, -0.50927734375, -2.357421875, -1.896484375, -1.5859375, -0.244140625, 0.310302734375, 0.54150390625, -1.5712890625, -0.487060546875, -0.1837158203125, 0.02838134765625, 2.80859375, -0.9794921875, -0.734375, 0.08544921875, 0.032958984375, -0.6943359375, -0.607421875, 0.421630859375, 0.79345703125, 2.400390625, 0.486572265625, 0.9423828125, -0.8369140625, 1.4697265625, 0.09722900390625, -0.3427734375, 1.4365234375, -0.70068359375, -0.61572265625, -0.14892578125, 0.0850830078125, -0.61376953125, -1.1884765625, -1.1728515625, -0.09454345703125, 0.27197265625, 0.87158203125, 0.09710693359375, -0.212646484375, -0.380126953125, -2.59375, 0.9775390625, -0.07757568359375, -0.37109375, 2.3984375, 0.4853515625, -0.7626953125, -0.13232421875, -1.931640625, -1.2392578125, -1.427734375, 0.3046875, 0.16162109375, 1.509765625, -0.94677734375, 0.552734375, -2.01953125, 1.4619140625, 1.861328125, 0.286865234375, -0.94775390625, -1.125, 1.1005859375, -1.2978515625, 0.609375, 0.3125, 1.0263671875, -0.79443359375, -0.67138671875, 1.09765625, 0.1585693359375, -1.1015625, -0.95361328125, -1.291015625, 1.0712890625, -0.1986083984375, 0.015228271484375, -0.1771240234375, -0.369140625, -1.916015625, 1.08984375, 0.6015625, 0.94873046875, -0.417236328125, 1.1240234375, -0.2467041015625, -0.96533203125, -0.541015625, -1.1240234375, 0.62109375, 0.892578125, -0.8671875, 0.583984375, 1.9794921875, -1.0927734375, -1.9609375, -0.62646484375, 0.089599609375, 0.85107421875, -0.36767578125, -2.587890625, 0.576171875, 1.208984375, 0.106201171875, -0.513671875, -1.078125, -1.04296875, 1.7578125, -0.71240234375, 0.4228515625, 1.7666015625, 0.4033203125, 1.458984375, 0.349609375, -0.9580078125, -0.398193359375, 0.13037109375, 1.88671875, -0.095458984375, -0.040771484375, 0.50732421875, -0.53369140625, -1.1640625, -0.2352294921875, 0.455322265625, -1.19140625, 0.98095703125, -1.2021484375, 0.56005859375, -1.70703125, 0.2734375, -0.8720703125, 0.935546875, 0.71435546875, 1.5693359375, 0.4111328125, 1.330078125, -1.2158203125, 0.736328125, 0.352783203125, 2.04296875, 0.487548828125, 0.01397705078125, -0.057220458984375, 0.1898193359375, 0.935546875, -0.327880859375, -0.75341796875, -0.1702880859375, 0.239013671875, -2.70703125, -0.300048828125, -0.23828125, 3.927734375, -0.69873046875, -0.181884765625, 1.91796875, 0.035675048828125, 1.1220703125, -1.67578125, -0.263916015625, 0.484619140625, -1.189453125, 0.0109100341796875, -0.01183319091796875, -0.242431640625, 0.90869140625, -0.70556640625, -1.5498046875, 0.8525390625, -0.44873046875, -0.390380859375, 1.3740234375, -0.385986328125, 0.92724609375, -0.1683349609375, -1.5029296875, -0.93212890625, 0.218505859375, 0.72802734375, -0.2344970703125, 2.37890625, 0.1329345703125, -0.2149658203125, -1.80078125, -0.42333984375, 0.0589599609375, -1.1455078125, 0.326904296875, -0.91845703125, 2.173828125, 2.36328125, 0.01508331298828125, 0.35107421875, 1.345703125, -0.6123046875, -1.0576171875, -0.176513671875, -0.63134765625, -1.3349609375, 1.76171875, 0.55224609375, -1.123046875, -0.1546630859375, -0.0055999755859375, -0.7001953125, 0.03485107421875, -0.0518798828125, -1.9580078125, -1.1669921875, -0.3427734375, 1.455078125, -0.296875, -0.7587890625, -0.458740234375, -0.396240234375, 1.529296875, -0.043731689453125, 0.3212890625, 1.4599609375, 0.884765625, 3.181640625, -1.365234375, -0.270263671875, -0.458251953125, -0.308837890625, -0.1783447265625, 1.3720703125, 0.8505859375, -0.60400390625, 1.322265625, -1.76953125, 1.4267578125, -0.8671875, 0.4833984375, -0.259521484375, -0.70947265625, 1.76171875, -0.25244140625, 0.09320068359375, -0.0762939453125, 0.164306640625, 1.6171875, 0.578125, 1.0302734375, -0.64111328125, 0.76318359375, 1.6083984375, 0.6171875, 0.64599609375, -0.408203125, 0.35791015625, -0.732421875, -1.01953125, 1.173828125, 1.1796875, -0.11602783203125, 1.283203125, -0.75537109375, 0.3046875, -1.294921875, 1.51171875, 0.8310546875, -0.818359375, -0.5087890625, -0.89501953125, 1.0830078125, -0.66357421875, 0.389892578125, 0.389404296875, 0.6240234375, -0.221923828125, -0.6162109375, -0.456298828125, 0.9638671875, 0.0016613006591796875, -0.0157318115234375, 0.265380859375, 1.0732421875, 0.958984375, -0.73095703125, 0.453857421875, 1.041015625, -0.419921875, 0.08258056640625, 1.068359375, 1.248046875, -0.7705078125, 0.806640625, -1.0771484375, 1.7099609375, -1.421875, -0.404296875, 0.301025390625, 0.79296875, -1.517578125, 0.6416015625, -2.46484375, 1.4736328125, -0.159423828125, 0.71728515625, 1.5576171875, -0.1558837890625, -1.4755859375, -0.30908203125, -0.235595703125, 1.8427734375, -0.70654296875, 0.11358642578125, 0.51171875, 0.33544921875, -0.66064453125, -0.80810546875, -0.78955078125, 0.59033203125, 1.44921875, -0.1907958984375, -1.2001953125, 1.0146484375, 1.3388671875, 0.88916015625, 0.8330078125, -0.39990234375, 1.5703125, 1.8623046875, -0.1884765625, -0.7490234375, -0.57666015625, -0.8671875, 0.72705078125, 0.552734375, -0.10455322265625, -0.67529296875, 0.0345458984375, -1.146484375, -0.1322021484375, -2.533203125, 0.6728515625, -1.3291015625, -0.42724609375, -0.58740234375, 0.0213470458984375, 0.71875, -0.1641845703125, -0.083984375, 0.70361328125, -1.26953125, 0.41552734375, -0.237548828125, 0.55712890625, 0.4765625, -0.8623046875, 0.90234375, 1.509765625, -0.72021484375, 0.58203125, 0.51171875, -0.04962158203125, 1.1083984375, 1.345703125, -0.32275390625, -0.0372314453125, -0.190185546875, -0.771484375, -0.94873046875, -0.41015625, 1.2236328125, -0.266357421875, -0.12408447265625, -0.30029296875, 0.20654296875, -0.3974609375, 0.308349609375, 2.166015625, -0.90625, -0.65771484375, -0.62109375, -0.7529296875, -0.54248046875, 0.030303955078125, 0.400390625, -1.3564453125, 1.369140625, 0.8310546875, -1.2197265625, -0.1943359375, -0.083251953125, -2.322265625, 0.69384765625, -0.411376953125, 1.02734375, -0.1788330078125, 0.3232421875, 1.3486328125, -0.76611328125, 0.7802734375, 0.63427734375, -0.279541015625, -0.7841796875, -1.234375, -1.8955078125, 2.033203125, 0.83984375, 0.07904052734375, -0.463134765625, -0.71630859375, -0.6083984375, -0.89306640625, -1.1796875, -0.490966796875, -0.053436279296875, -0.13818359375, -0.62109375, 0.482421875, -0.41357421875, 1.2177734375, -0.5966796875, -1.7607421875, 1.7744140625, 0.00276947021484375, 1.0263671875, -0.1322021484375, -0.6171875, -0.162109375, 0.35302734375, -0.9208984375, -0.44921875, -0.10394287109375, 0.69775390625, 0.66943359375, -0.407958984375, -0.489501953125, -0.84716796875, 0.859375, 0.374755859375, 0.69482421875, 0.935546875, 0.8935546875, -0.70703125, 0.8603515625, -0.50244140625, -0.79248046875, -1.16796875, -0.5888671875, -1.5634765625, 0.1846923828125, 0.26416015625, -0.46875, 0.401123046875, 0.0609130859375, -1.4658203125, 0.25634765625, 0.7734375, -0.63134765625, 0.481689453125, -0.56201171875, -0.9697265625, -0.05364990234375, 0.8759765625, 0.75537109375, -0.1904296875, -0.55419921875, -1.4873046875, -0.5263671875, 1.1162109375, 0.83447265625, -0.1695556640625, 1.4013671875, 1.7314453125, 0.80859375, 0.55810546875, -1.115234375, 0.405029296875, -0.9892578125, 1.673828125, 2.333984375, 0.12646484375, 0.367919921875, 1.4287109375, -0.4716796875, 2.244140625, -1.4541015625, 0.53564453125, -0.217529296875, 0.51171875, 0.3974609375, 0.51611328125, 0.54541015625, -1.5283203125, 0.6904296875, 0.2008056640625, -1.3154296875, -1.244140625, -0.6494140625, -0.2396240234375, -1.28515625, -0.0192718505859375, 2.23828125, -0.5302734375, 1.751953125, -0.81103515625, -0.56591796875, 0.38037109375, 0.6328125, -0.29248046875, 0.65966796875, -0.59814453125, -0.2113037109375, -0.76123046875, 1.0478515625, 0.26416015625, -0.65869140625, -1.2470703125, 0.25927734375, 0.1734619140625, 0.0723876953125, -0.2281494140625, 1.3779296875, -1.8818359375, -0.30029296875, -0.5126953125, -0.444580078125, 0.1925048828125, 0.52099609375, 1.103515625, 1.16796875, -1.701171875, -0.67431640625, -1.251953125, 0.11468505859375, -2.26953125, -0.9130859375, 0.0787353515625, 1.1396484375, -2.59765625, 1.0048828125, -0.358154296875, -0.04217529296875, 1.3623046875, 1.0322265625, -0.229248046875, -0.75, 0.66015625, 0.427001953125, -0.459716796875, 1.130859375, -0.1251220703125, -0.380126953125, 0.689453125, -1.533203125, -1.884765625, 0.16015625, 1.109375, 0.54345703125, 0.8232421875, -1.384765625, 0.83642578125, -1.1220703125, 0.2255859375, -1.3310546875, -0.278076171875, 0.0911865234375, -0.11517333984375, 0.0738525390625, 1.1318359375, 0.6708984375, -0.5615234375, -2.076171875, 0.86669921875, 1.017578125, 1.2646484375, -0.001983642578125, 1.279296875, 1.0439453125, 1.458984375, -0.87060546875, 0.64306640625, 0.035980224609375, -0.77099609375, 0.167724609375, -1.3466796875, 0.63427734375, -0.14892578125, -1.5751953125, 1.02734375, -0.0189971923828125, 0.2449951171875, -1.537109375, 0.2274169921875, -0.048675537109375, -1.6865234375, -0.54931640625, -0.07135009765625, -0.5517578125, -2.224609375, -1.2841796875, -0.0362548828125, -0.78515625, 1.033203125, 1.2021484375, 1.2978515625, 0.59033203125, 2.142578125, -0.0252227783203125, -2.01171875, 1.4091796875, 0.2841796875, 0.07110595703125, 0.8994140625, -0.4345703125, -1.068359375, 0.62890625, -1.609375, -0.7529296875, 1.1865234375, -0.62890625, -0.98046875, 0.50341796875, 0.06396484375, 1.15234375, -1.2392578125, 0.8984375, -0.10638427734375, 1.666015625, -0.403076171875, -1.9306640625, -0.08111572265625, 0.049224853515625, 0.755859375, 1.12890625, -0.37158203125, -1.541015625, -0.55859375, -0.197998046875, 0.496337890625, -0.67333984375, -0.5546875, -0.02587890625, 0.05902099609375, -0.64697265625, -0.4677734375, 0.255859375, -1.9345703125, 0.266357421875, 0.44189453125, -1.0537109375, -0.6533203125, -0.2099609375, 0.08740234375, -3.3828125, -0.423583984375, 2.2109375, 1.822265625, 0.368896484375, -1.193359375, -0.701171875, 0.1517333984375, 0.96923828125, 0.94970703125, 0.36376953125, 0.75341796875, 0.34375, -0.22900390625, 0.90966796875, 1.6904296875, -0.482666015625, 1.8857421875, -1.7568359375, 0.19775390625, 1.2724609375, 0.2021484375, 1.021484375, 1.064453125, 0.27294921875, -0.580078125, -1.3798828125, -0.22265625, 0.307373046875, 0.484375, 0.042327880859375, 0.1510009765625, 1.8759765625, -1.6455078125, 1.263671875, -1.431640625, -1.0146484375, 0.60595703125, -1.873046875, 0.27734375, -0.67822265625, -1.5078125, -3.021484375, 0.58740234375, -0.06414794921875, 0.293701171875, 1.158203125, -0.10321044921875, -1.3154296875, 0.115234375, 0.1102294921875, -0.0189971923828125, -0.65771484375, 2.396484375, -0.74462890625, -2.0625, -0.7294921875, -0.720703125, 0.07916259765625, -0.310546875, -1.1630859375, -0.6494140625, 0.48828125, 0.8544921875, 0.47900390625, 0.46923828125, 1.3056640625, -1.4462890625, 0.07489013671875, -0.018218994140625, 0.32421875, 0.442626953125, -1.138671875, -1.2509765625, -0.07318115234375, 0.26513671875, 0.98388671875, 1.0810546875, -0.026153564453125, 2.576171875, -0.8662109375, -1.421875, 1.8134765625, 0.12353515625, -1.3271484375, -1.1884765625, 1.3505859375, -0.3447265625, -0.51416015625, -2.529296875, 0.05126953125, -0.294921875, 0.8359375, -0.470458984375, 1.2392578125, 0.1536865234375, 1.5322265625, 0.9736328125, 1.1953125, 0.99609375, 0.184326171875, 0.85791015625, -1.634765625, 1.1728515625, -0.80517578125, 0.73974609375, -0.228271484375, -0.443115234375, -0.70654296875, -0.15380859375, -1.0185546875, -0.1341552734375, -1.1298828125, 1.7216796875, -1.525390625, 0.94580078125, -1.0673828125, 0.9072265625, -0.72314453125, -1.1083984375, 1.326171875, 0.8310546875, 0.428955078125, -0.73974609375, 0.56884765625, 0.283935546875, -1.150390625, -0.490234375, 1.4287109375, -0.6669921875, -1.26171875, -1.1337890625, -0.330078125, 0.77734375, -0.7607421875, 0.83349609375, -0.314697265625, -0.0201568603515625, -0.548828125, 0.279296875, -0.05059814453125, 1.30078125, 1.41015625, -1.5146484375, -1.931640625, -0.1710205078125, -0.68798828125, -0.1239013671875, 0.386962890625, -1.90234375, 0.146728515625, 0.72021484375, 0.2578125, 0.05706787109375, 0.7236328125, -0.01239776611328125, 0.45068359375, -0.023712158203125, -0.1365966796875, 0.529296875, 0.8662109375, -0.7509765625, -1.0849609375, -0.92626953125, 0.083984375, 0.328125, 0.689453125, -0.1451416015625, 1.0234375, -0.65869140625, 0.74169921875, 0.34033203125, -0.069091796875, -1.4619140625, 0.64599609375, 1.4609375, -0.28955078125, 0.89599609375, 3.0234375, -0.99853515625, -0.2861328125, -0.97021484375, -0.9541015625, -0.259765625, -0.362060546875, -1.5927734375, -1.505859375, -2.48828125, -0.2430419921875, -0.0771484375, 0.0300445556640625, 2.337890625, 1.8720703125, 0.445556640625, 1.3115234375, 1.142578125, -1.306640625, 0.5029296875, -1.6484375, 1.5517578125, -0.9072265625, 0.0054473876953125, 0.226806640625, -0.818359375, 0.0855712890625, -1.1162109375, 0.74853515625, 0.489013671875, 0.73828125, -1.6728515625, 1.2890625, 0.370361328125, -0.9755859375, 0.50390625, -1.4951171875, 0.6455078125, 0.252197265625, 0.7236328125, 0.55126953125, -1.431640625, 1.396484375, 1.5205078125, -1.6923828125, -0.8193359375, -2.5078125, -1.6611328125, 0.198974609375, -1.0458984375, 0.43505859375, 1.6357421875, -0.51171875, 1.0771484375, 1.818359375, -1.4267578125, 1.2216796875, -0.026275634765625, 0.08050537109375, 0.366943359375, -0.314697265625, -1.3603515625, 0.84619140625, -0.60693359375, -1.021484375, -1.0625, -0.4658203125, 0.56298828125, -1.5009765625, 0.22216796875, 0.8671875, -0.50537109375, 0.2288818359375, -0.281005859375, 0.418212890625, -0.4775390625, -0.82666015625, -0.2744140625, -1.05859375, -0.49658203125, 1.0634765625, -0.82861328125, 0.99658203125, 0.93017578125, -0.414306640625, -0.05670166015625, -0.07196044921875, 2.44921875, -0.2115478515625, 0.20947265625, 0.5048828125, -0.76513671875, 0.8544921875, 0.41162109375, -0.61962890625, 0.8486328125, 0.619140625, -0.0281982421875, 0.328369140625, -0.72900390625, 0.6572265625, -0.157470703125, 0.386962890625, 0.271240234375, -0.146484375, -1.40234375, 0.29931640625, 2.193359375, 1.3759765625, 0.94873046875, -0.439208984375, -0.2890625, 0.182373046875, 0.257080078125, -2.07421875, 1.1298828125, 0.68212890625, 0.85400390625, -1.7978515625, 0.478515625, 0.474853515625, -0.348388671875, 0.6376953125, 0.099365234375, -0.9365234375, 1.6015625, -1.5751953125, 0.3740234375, 0.99658203125, -2.078125, -0.073486328125, -0.469482421875, -0.3701171875, -0.64697265625, -0.341796875, 0.287353515625, -1.3369140625, 0.203125, 0.99267578125, -1.8076171875, 0.2069091796875, -0.9921875, -0.0650634765625, -0.81103515625, 0.482421875, 1.0546875, 0.59521484375, -1.28515625, 0.152099609375, -0.2425537109375, 0.082763671875, -0.75048828125, -0.81689453125, -0.176025390625, -0.18408203125, -1.0302734375, 0.7685546875, 0.9580078125, -1.3837890625, 0.24658203125, 0.6533203125, -1.8388671875, -0.15576171875, 0.47802734375, 0.34912109375, 1.1416015625, -1.3447265625, 0.256103515625, 0.0528564453125, -2.296875, 1.8525390625, 1.2578125, -0.1268310546875, -0.7353515625, -0.38671875, -0.486328125, -0.7509765625, -1.3994140625, 0.7685546875, -0.496337890625, -0.94482421875, -0.2147216796875, 1.203125, -0.2022705078125, -1.5048828125, -0.58447265625, -1.552734375, -0.8662109375, -2.34765625, 1.5302734375, 0.98779296875, 0.40478515625, -1.5087890625, 0.83935546875, 1.333984375, -0.39990234375, -1.3408203125, 1.4013671875, 0.302978515625, -0.71240234375, 2.373046875, -1.4580078125, 0.362548828125, 0.374755859375, 1.849609375, 0.414306640625, -0.796875, 2.154296875, 1.8017578125, 0.0943603515625, -1.736328125, 0.865234375, -0.00428009033203125, -2.09765625, -0.67431640625, -0.1513671875, -0.64208984375, -2.244140625, 0.72705078125, -0.2205810546875, -0.4794921875, 1.701171875, 0.59033203125, 0.8330078125, -0.3916015625, -0.7841796875, 0.46240234375, -1.33984375, -0.3544921875, 0.7587890625, 0.1788330078125, -0.42578125, 0.1800537109375, 1.013671875, 0.74658203125, -1.193359375, -0.451416015625, 0.18798828125, 0.43505859375, 0.05792236328125, -0.178955078125, -0.63671875, -0.69482421875, 0.80126953125, 1.51953125, -0.1400146484375, -1.0439453125, 0.947265625, 0.02294921875, -0.521484375, -0.125244140625, -1.2734375, 0.7919921875, 1.126953125, -1.291015625, 1.59765625, 0.77880859375, 1.7529296875, -0.3212890625, -2.1171875, -0.3837890625, 0.58154296875, 2.03125, 0.409912109375, -0.380615234375, 0.1256103515625, -1.4580078125, -0.7578125, -2.4140625, 0.351318359375, 0.00811004638671875, -0.61572265625, 0.515625, -0.496337890625, 0.8984375, 1.3056640625, 1.4521484375, -0.814453125, -1.2646484375, 0.345947265625, -1.806640625, 0.93212890625, 0.7841796875, -1.458984375, -0.2103271484375, -1.236328125, 0.1942138671875, -0.32763671875, -1.54296875, 0.1761474609375, 0.0170135498046875, -1.21484375, 1.2470703125, 1.255859375, 1.0859375, 1.0361328125, 2.3359375, -1.384765625, -0.55126953125, 1.2724609375, -1.39453125, 1.037109375, 1.1220703125, 0.07659912109375, -0.72265625, 0.72314453125, 1.2607421875, 0.354248046875, 0.29931640625, -0.08172607421875, 0.830078125, 1.3076171875, 0.5244140625, 1.421875, -2.673828125, -1.474609375, -1.2138671875, 0.450927734375, 2.142578125, 0.35888671875, -1.32421875, 0.3701171875, 0.53564453125, -1.0205078125, 0.07684326171875, -1.4912109375, -0.077880859375, 0.36669921875, 0.082275390625, -0.978515625, 0.330322265625, -0.88134765625, -0.338623046875, -0.193115234375, 0.0762939453125, -1.08984375, -1.0517578125, -0.8720703125, 0.42138671875, 0.6005859375, 1.490234375, -1.2822265625, 0.232177734375, -1.0947265625, -0.0966796875, 0.1490478515625, -1.4560546875, -0.72119140625, 0.1689453125, 0.21728515625, -1.390625, 0.1903076171875, -0.9541015625, -0.3701171875, 0.9189453125, 0.60302734375, -0.32177734375, 1.255859375, 0.8046875, -0.9404296875, 0.65283203125, -1.0283203125, 0.2332763671875, -1.09375, 0.75244140625, 0.791015625, 0.1876220703125, 1.681640625, -1.11328125, 0.046630859375, -1.896484375, -0.0242156982421875, -1.119140625, -0.1580810546875, 1.0810546875, -0.281005859375, -0.414794921875, 0.375, -1.9833984375, 0.37548828125, 1.1337890625, -1.6728515625, -1.1484375, 1.0712890625, -1.5537109375, 0.4453125, 1.8955078125, -0.9501953125, 0.1536865234375, -0.75830078125, -0.90087890625, -0.1683349609375, -1.63671875, 1.2939453125, 1.5751953125, -0.6474609375, 0.61474609375, 0.5849609375, 1.3837890625, -2.30859375, 1.47265625, 1.0087890625, 0.96142578125, -0.78662109375, 0.31103515625, -0.50830078125, 0.8349609375, 0.712890625, -0.69873046875, 0.6064453125, -1.30859375, 0.708984375, 0.33203125, 0.73779296875, -0.76904296875, 0.1483154296875, -0.6767578125, 0.93505859375, 1.3857421875, -1.787109375, -0.295654296875, 0.83544921875, -0.3828125, -0.80126953125, -0.005458831787109375, -0.2039794921875, -0.348388671875, 0.004108428955078125, 0.346923828125, 1.18359375, -1.541015625, -0.45751953125, -0.303955078125, 1.427734375, 0.90478515625, -0.5322265625, 0.5595703125, 1.6474609375, -2.048828125, -0.87646484375, -0.55517578125, 1.7041015625, 0.92724609375, 0.8994140625, -0.265625, 1.7470703125, -0.415771484375, 0.4365234375, 0.85888671875, 1.2841796875, -1.0341796875, 0.244873046875, -0.16650390625, -1.6455078125, 0.207763671875, -0.396728515625, -0.358154296875, -1.3359375, 1.1318359375, 0.2529296875, -2.373046875, 1.517578125, 0.79931640625, 0.57177734375, -0.8427734375, -0.662109375, 1.2021484375, 1.6484375, 0.79541015625, -0.7060546875, -1.2021484375, -0.1363525390625, 0.7236328125, 0.9765625, 0.763671875, 0.6005859375, 0.55712890625, 0.60498046875, -0.67822265625, 0.79052734375, -0.94921875, 0.000644683837890625, -1.2138671875, 2.421875, -0.0163116455078125, -0.343505859375, 0.9150390625, 0.564453125, -0.380615234375, 0.1744384765625, -1.1884765625, -0.99267578125, -1.4462890625, -0.990234375, -0.6005859375, 0.250244140625, 0.331787109375, -0.3642578125, -1.134765625, 0.55810546875, 2.634765625, 0.9150390625, 0.219482421875, -0.505859375, 0.97705078125, 0.363525390625, -0.93994140625, -2.677734375, 0.1094970703125, -0.70751953125, -0.51513671875, 0.1611328125, -0.85205078125, -0.724609375, -0.270751953125, 1.20703125, -0.78564453125, -0.168701171875, -0.325439453125, -0.11053466796875, -0.81005859375, -0.80908203125, -0.5703125, 0.35107421875, -0.1954345703125, 0.4384765625, 1.12109375, 1.04296875, 1.5791015625, -0.7978515625, 0.35400390625, 1.013671875, 0.44677734375, 1.2568359375, -1.3681640625, 0.80224609375, 1.599609375, -1.513671875, -1.4404296875, 1.236328125, -0.455810546875, -0.04827880859375, 0.91015625, -0.2447509765625, -1.1474609375, 0.57080078125, 0.52001953125, 0.2481689453125, 0.278076171875, -0.8671875, -0.298583984375, -1.2109375, -0.033660888671875, -0.54296875, -0.55078125, 1.8173828125, -0.4013671875, 1.5439453125, 0.64111328125, -1.4482421875, -0.331298828125, -0.73291015625, 0.8671875, 0.275146484375, 0.439697265625, 1.4990234375, -0.4892578125, 0.258544921875, -1.95703125, 0.75732421875, 0.71826171875, 0.62451171875, 0.1741943359375, 0.285888671875, -0.67138671875, -0.62841796875, 0.70654296875, 0.52734375, -0.453369140625, 0.95458984375, 1.13671875, -0.092529296875, -0.138671875, 1.84375, 0.2451171875, 0.61279296875, 0.208740234375, 2.09375, -0.421630859375, 0.268798828125, -0.80712890625, 0.060333251953125, 0.7763671875, 0.1463623046875, 0.86669921875, -1.392578125, 0.343017578125, 0.1370849609375, 0.8349609375, -0.423583984375, -0.697265625, -2.580078125, -1.7626953125, -0.60546875, -0.270263671875, 0.3330078125, -0.41357421875, -1.7490234375, -0.564453125, -1.5478515625, 1.7578125, -2.525390625, -0.8466796875, -1.1591796875, -0.6318359375, 1.103515625, 0.85107421875, -0.6904296875, -0.951171875, -0.77880859375, 0.07708740234375, -1.697265625, -2.6796875, -0.6708984375, -0.6669921875, -1.9306640625, 0.036163330078125, 0.29541015625, 2.0, 1.0830078125, -0.12359619140625, 0.450439453125, -0.5224609375, 0.3046875, -0.471923828125, -0.1832275390625, -0.0004525184631347656, -0.335693359375, -0.9267578125, -0.51025390625, -0.66552734375, -0.85986328125, -2.107421875, -1.6708984375, 1.8271484375, -2.10546875, -1.4208984375, -0.63916015625, 0.047393798828125, 3.015625, -1.3115234375, -0.452392578125, 1.5673828125, -0.65234375, -1.37890625, -0.0081939697265625, 1.7763671875, -0.02667236328125, 0.462646484375, -0.0458984375, 1.3251953125, 0.06805419921875, -0.2398681640625, -2.890625, -1.984375, -0.68603515625, 0.84033203125, 0.22998046875, -1.93359375, -0.7236328125, 0.4619140625, 0.4169921875, 0.354248046875, -1.2841796875, -1.2978515625, 1.4072265625, -0.343994140625, -0.55419921875, -1.0654296875, -0.224609375, -1.3798828125, -1.5166015625, -0.2183837890625, 0.4384765625, 1.9443359375, 0.48974609375, 0.82666015625, -1.4091796875, -1.078125, 2.484375, -0.8681640625, -0.197509765625, -1.443359375, 0.54443359375, 1.3935546875, -0.493408203125, -0.2230224609375, -1.0478515625, 0.2078857421875, -0.62890625, -2.263671875, -1.9208984375, 1.39453125, 1.556640625, 0.1678466796875, -0.5712890625, 0.7177734375, 0.1361083984375, 0.1436767578125, -0.92333984375, 0.1224365234375, -1.9990234375, 0.264892578125, 1.890625, 1.38671875, 0.77294921875, -0.138671875, -0.95849609375, 0.79638671875, 1.2177734375, 0.5224609375, -0.6162109375, 0.324462890625, -1.74609375, 0.1158447265625, -0.876953125, 0.254638671875, -0.3134765625, -1.671875, -0.494384765625, -0.6875, -0.07464599609375, 1.4609375, -0.76123046875, -0.88037109375, 0.9267578125, -0.6962890625, 0.346435546875, -0.7890625, -0.1142578125, 1.658203125, -1.30078125, 0.0838623046875, 0.41796875, -0.366455078125, -1.4443359375, 0.461181640625, -0.0633544921875, -1.0078125, -1.1142578125, 0.34423828125, -0.83203125, 1.7744140625, -0.0312042236328125, 0.5048828125, -1.2109375, 0.611328125, 1.0576171875, 0.06243896484375, 0.482666015625, -0.7802734375, 0.294921875, 1.6552734375, 0.0914306640625, 2.095703125, 1.353515625, 0.186767578125, -0.748046875, 0.2744140625, -0.136962890625, 0.061309814453125, -0.8916015625, 0.326171875, 0.50048828125, -0.4072265625, 0.350341796875, -0.89892578125, 0.332763671875, 0.428955078125, -0.3505859375, 0.775390625, -1.6015625, 0.17138671875, 0.1258544921875, 0.58740234375, 0.71630859375, 2.15625, -1.4892578125, -0.865234375, 0.724609375, 2.29296875, -1.13671875, -0.71484375, -0.0987548828125, 2.068359375, 0.035552978515625, 0.025604248046875, 0.059326171875, -1.2431640625, -0.2476806640625, -0.63232421875, 0.5009765625, 0.40625, -0.14697265625, -0.572265625, 0.32421875, -0.765625, 1.2138671875, -1.0966796875, -0.7568359375, -0.03521728515625, -1.0166015625, -0.57080078125, 1.123046875, -1.2060546875, 1.3037109375, -0.7578125, 1.2822265625, -0.19384765625, -0.10357666015625, -0.7333984375, -0.873046875, 0.7802734375, 0.27734375, -0.2081298828125, -0.013885498046875, 2.3828125, 0.4150390625, 1.91015625, 1.5244140625, -0.485107421875, 0.9365234375, 1.037109375, -0.144775390625, 0.361083984375, -1.486328125, -1.1728515625, 1.625, -0.206298828125, -0.1641845703125, 0.6767578125, 0.8427734375, 0.06378173828125, -0.923828125, -0.398193359375, 0.9794921875, 1.3818359375, 0.39208984375, 0.59130859375, 1.8447265625, -0.487548828125, -0.70263671875, -1.1357421875, 1.2265625, -1.88671875, 1.1875, 0.02752685546875, 0.09698486328125, -0.22021484375, 0.001190185546875, 0.99853515625, 1.279296875, 1.1142578125, 0.475341796875, -0.2171630859375, -2.1796875, -1.4384765625, 0.88330078125, 0.7265625, 0.97216796875, 1.548828125, 0.716796875, 1.3662109375, -2.228515625, -0.8974609375, 0.30419921875, -0.464599609375, -0.303955078125, -0.406005859375, -0.57958984375, 0.11871337890625, 0.31494140625, -0.798828125, -2.2890625, -0.050323486328125, -1.607421875, 0.697265625, 1.38671875, 1.87109375, 0.339599609375, -0.427734375, -2.169921875, 0.79541015625, -0.908203125, 0.417236328125, 0.21337890625, -0.482421875, 0.0806884765625, -0.07635498046875, 0.99365234375, -0.96533203125, 0.381103515625, -0.57421875, -0.513671875, -0.1409912109375, -0.228759765625, 0.52099609375, 0.476318359375, 0.39697265625, 0.3828125, 1.853515625, 2.07421875, 3.8828125, -1.9814453125, 0.68212890625, -0.2039794921875, -0.8173828125, 0.3330078125, -0.7490234375, 0.3935546875, 0.58154296875, 0.270751953125, 0.47705078125, -0.426513671875, 0.5341796875, 0.09393310546875, -0.7568359375, 0.8046875, -0.002239227294921875, -0.51806640625, 1.107421875, 1.9189453125, 2.060546875, 1.4716796875, 0.29638671875, 0.123291015625, -2.4375, 2.068359375, 0.70458984375, 0.387451171875, -0.5634765625, 1.3505859375, 0.1063232421875, -1.634765625, 0.76220703125, 0.17822265625, -0.52490234375, 2.330078125, -0.1483154296875, -1.0634765625, 0.358642578125, 0.50634765625, 1.0205078125, 0.62841796875, 0.30810546875, -0.08526611328125, -1.4560546875, 1.609375, -0.29541015625, -0.050079345703125, 1.5380859375, -3.744140625, -1.5107421875, -0.94580078125, -0.92236328125, -0.15478515625, -0.1822509765625, 0.5546875, 1.0986328125, -0.6416015625, 0.890625, -0.5595703125, 2.505859375, 0.46630859375, 0.03839111328125, 0.23388671875, 0.002147674560546875, -0.06671142578125, -0.033660888671875, 0.79345703125, 0.24951171875, -0.9169921875, 1.6953125, -1.28125, 1.5234375, -0.5205078125, 0.07470703125, -1.8125, -1.044921875, 0.7060546875, -0.3974609375, 0.08642578125, -0.038818359375, 1.5654296875, 1.5283203125, 0.626953125, 0.01003265380859375, -0.2469482421875, -1.48046875, 0.4169921875, 1.1162109375, 0.19091796875, 0.83447265625, 1.1884765625, -0.479736328125, 0.55224609375, 1.50390625, 1.16796875, -0.75830078125, -1.2607421875, 1.490234375, -0.12420654296875, -0.1036376953125, -0.611328125, 0.98046875, -0.64453125, 0.13671875, 0.396484375, 0.406494140625, -0.85498046875, -0.98876953125, 1.3330078125, -0.876953125, -0.0814208984375, -0.006927490234375, -1.091796875, 0.8408203125, -0.81640625, 0.6044921875, -1.572265625, -0.515625, 0.87255859375, 1.1142578125, 0.68798828125, 0.290771484375, -0.363037109375, 2.04296875, -1.4228515625, -0.1649169921875, -1.43359375, -2.259765625, 0.724609375, 1.6181640625, -0.8544921875, -0.27001953125, -0.93212890625, -0.30859375, -0.6181640625, 0.705078125, -0.1986083984375, 0.61767578125, -0.35400390625, -1.1669921875, 0.285400390625, 0.56494140625, 0.45361328125, -0.2388916015625, -0.7431640625, 0.767578125, 0.70947265625, 0.9111328125, 0.54541015625, 0.58251953125, 0.9296875, 1.1103515625, -0.1632080078125, 0.212890625, 1.3427734375, -0.974609375, 1.3310546875, -1.0048828125, 0.728515625, 0.50732421875, -0.01255035400390625, 1.1015625, 0.904296875, 1.056640625, 0.26123046875, -1.9248046875, -0.9228515625, 0.630859375, -0.64990234375, -1.162109375, -0.06646728515625, 0.03179931640625, -2.7578125, -0.751953125, 0.044647216796875, 0.01494598388671875, 0.97607421875, 0.433837890625, -0.66064453125, -1.59375, 0.275390625, -0.736328125, 0.71923828125, -1.8603515625, 0.56689453125, -0.7451171875, 0.2218017578125, -0.95556640625, 0.300048828125, 2.396484375, 0.197021484375, -0.30859375, -0.39306640625, 0.95556640625, -0.560546875, 1.2802734375, 0.275634765625, 0.06903076171875, 1.09765625, 1.451171875, -0.1656494140625, -1.287109375, -0.048370361328125, 0.291748046875, -0.82177734375, -0.86083984375, -0.67041015625, -1.60546875, -1.3564453125, 3.1484375, 1.5478515625, -0.5966796875, -1.875, 0.06927490234375, -1.064453125, -1.3505859375, 0.252685546875, 0.94189453125, 0.201904296875, -0.564453125, -1.0205078125, -0.80908203125, 1.775390625, -2.2421875, -0.93115234375, -0.080078125, -0.34619140625, -0.98681640625, -0.7626953125, 0.348388671875, -0.78076171875, -0.30908203125, 0.146240234375, -1.310546875, 0.37353515625, 0.7509765625, 1.1650390625, 0.58984375, -0.2393798828125, 1.19140625, -0.50927734375, -1.0068359375, 0.6650390625, 0.86083984375, 1.6748046875, -1.525390625, 0.9296875, 1.3681640625, 0.437255859375, -0.3505859375, 0.80126953125, 0.59765625, -0.446533203125, -0.82763671875, 1.5126953125, 1.10546875, 0.433837890625, 0.3564453125, -0.92919921875, -0.38916015625, 0.456298828125, 1.1552734375, 0.38623046875, -1.8056640625, -1.0234375, -0.765625, -1.7060546875, 0.014434814453125, -2.283203125, 0.160888671875, -0.035430908203125, -0.225341796875, -0.443359375, -0.3896484375, -1.7451171875, 0.406982421875, -1.74609375, -0.93603515625, -0.00601959228515625, 0.12091064453125, -1.7763671875, 0.74951171875, 0.4365234375, -1.2880859375, 0.453125, -2.0390625, 0.2418212890625, 1.1181640625, -1.1513671875, -1.4765625, 1.3173828125, -0.5458984375, -0.662109375, 0.716796875, 0.25537109375, -1.2509765625, 1.1484375, -0.08746337890625, 0.373779296875, -1.162109375, -0.130859375, 1.0693359375, -0.11236572265625, 0.25341796875, 1.1875, 0.2220458984375, 0.0222625732421875, -0.297119140625, 1.4951171875, 0.6064453125, -0.0260772705078125, -0.54541015625, -0.50537109375, 0.95263671875, -1.7646484375, -1.341796875, 0.0809326171875, -0.476806640625, -0.791015625, 0.4404296875, 0.74951171875, -2.5703125, 0.60302734375, 1.3271484375, -0.65966796875, 0.157958984375, -0.1759033203125, -0.310302734375, 0.68701171875, -0.34765625, 0.3017578125, -0.671875, 0.33349609375, -0.94921875, 0.1937255859375, 0.80224609375, 0.69287109375, 1.904296875, -0.138671875, 0.70556640625, -0.1409912109375, 0.68017578125, 0.365966796875, 0.69580078125, 0.6171875, 1.330078125, -0.5, 0.1871337890625, 0.79296875, -0.68408203125, -0.1551513671875, 0.14990234375, -2.318359375, 0.79345703125, 1.6240234375, 0.340576171875, 1.3291015625, 1.140625, 0.105224609375, 0.46142578125, 1.64453125, -2.322265625, -0.32373046875, 0.66259765625, -0.72509765625, -0.6435546875, 1.8896484375, -0.52001953125, 1.03125, 0.0024509429931640625, -1.1123046875, -0.71240234375, 0.962890625, 1.923828125, -1.1806640625, -0.89111328125, 1.568359375, 0.492919921875, 0.751953125, -0.5703125, -0.65673828125, 1.40234375, 0.09130859375, 1.392578125, -0.5283203125, -0.5146484375, 0.943359375, 0.210693359375, -0.365234375, 0.80615234375, 0.607421875, 0.202880859375, -1.41015625, 0.458740234375, -0.63232421875, 0.199462890625, -1.486328125, -0.10491943359375, 0.313720703125, -0.144287109375, -1.03515625, -2.349609375, -1.0263671875, 0.8291015625, -0.11865234375, -0.76171875, 0.7490234375, 1.5263671875, 0.386474609375, 1.5009765625, -0.65380859375, 1.1982421875, 0.580078125, 0.314697265625, -0.442626953125, -0.1669921875, -1.4228515625, 0.46484375, -0.6865234375, 1.0615234375, 1.1240234375, -1.021484375, -1.3642578125, 1.3486328125, 0.71923828125, -1.57421875, 0.59765625, 1.537109375, 2.005859375, 0.67822265625, 1.974609375, 0.493408203125, 0.24560546875, 0.259765625, 0.51220703125, 1.6494140625, -0.1759033203125, 0.6357421875, -0.69921875, 0.147216796875, 0.299560546875, -1.6298828125, -1.0078125, 0.15234375, -0.55712890625, 0.1829833984375, 0.049835205078125, -0.338134765625, -0.9130859375, 0.64453125, -0.211181640625, 0.310791015625, 0.460693359375, -0.137451171875, 0.65283203125, 0.24462890625, -0.5341796875, -1.1396484375, 0.031707763671875, -1.73046875, -0.376953125, -1.013671875, 0.8876953125, 0.1488037109375, 0.67919921875, 0.5283203125, -0.2100830078125, -1.13671875, 2.310546875, 0.384521484375, 0.85498046875, 1.26953125, 0.37939453125, -0.09423828125, 0.7919921875, -1.455078125, -1.958984375, -1.0185546875, 1.7138671875, 0.7578125, 0.45654296875, -2.755859375, 1.3720703125, -0.459716796875, 0.395751953125, -0.9580078125, 0.421630859375, -0.52490234375, 1.40625, -0.923828125, -0.75341796875, -0.939453125, 0.73046875, -0.472900390625, 0.5126953125, 0.11767578125, -0.5244140625, -0.70361328125, 0.6044921875, 0.5302734375, -0.85107421875, -1.8291015625, 1.185546875, -0.24755859375, 0.94775390625, -1.5244140625, -0.388916015625, -1.7783203125, 0.50146484375, -1.376953125, 0.417236328125, 0.042236328125, -0.9697265625, 0.7392578125, 1.19921875, -0.353759765625, 0.249755859375, -0.0611572265625, -0.42724609375, 1.3876953125, 0.4521484375, -0.96435546875, -0.421142578125, -0.107666015625, 0.035125732421875, 0.32568359375, -1.13671875, 1.1005859375, 0.29052734375, 1.587890625, -0.5322265625, -0.84130859375, 0.463134765625, -0.50830078125, -1.615234375, -0.2156982421875, -0.302490234375, -1.314453125, -0.314208984375, -1.1162109375, -0.288330078125, -0.67626953125, 1.5791015625, -0.07366943359375, -0.71142578125, -2.34765625, -1.0302734375, -1.39453125, -1.07421875, -0.54541015625, 0.111572265625, 0.779296875, 0.7412109375, -1.884765625, 1.177734375, -0.07135009765625, 0.1434326171875, 0.0268707275390625, -0.5966796875, -1.4072265625, 0.0188446044921875, 0.69482421875, -0.17431640625, -0.473876953125, -0.1448974609375, 0.41357421875, 1.171875, -0.94482421875, -0.0006737709045410156, -0.56640625, -0.380859375, 1.466796875, 1.0986328125, -1.390625, -0.287109375, 0.52099609375, 0.0728759765625, -1.1748046875, 0.330810546875, 0.493408203125, 0.9404296875, -0.463623046875, 0.219970703125, 0.61865234375, -0.01580810546875, 0.35693359375, 1.0625, 1.1064453125, 0.783203125, -1.33984375, 1.630859375, 0.98779296875, -0.759765625, 1.048828125, 1.7978515625, -0.39599609375, -0.314697265625, -0.319091796875, 0.767578125, -1.4150390625, 1.7548828125, -0.591796875, -0.66162109375, -0.1134033203125, -1.1494140625, 0.6689453125, -0.2073974609375, 2.5390625, -0.576171875, -0.1556396484375, 0.9267578125, -0.21630859375, -0.450927734375, 0.9375, 0.417724609375, 0.2279052734375, 0.6572265625, -0.62939453125, -1.16015625, -0.82373046875, -0.471435546875, -0.35400390625, 0.8251953125, 0.5859375, 0.07623291015625, 0.197509765625, 0.88037109375, -0.33203125, -0.7998046875, -0.3349609375, 0.3125, 2.0625, -0.427734375, -1.541015625, 0.943359375, 1.2060546875, 0.48486328125, -2.265625, -0.9833984375, 1.5869140625, 2.697265625, 1.1337890625, -0.5732421875, 0.2139892578125, 0.6865234375, 0.458740234375, -1.068359375, 0.9931640625, 0.1998291015625, -2.013671875, -3.466796875, -1.9912109375, -0.1572265625, -0.6865234375, 0.52587890625, 0.53564453125, 1.892578125, 1.55859375, 0.4189453125, 0.00506591796875, 0.412841796875, 0.2255859375, -1.6962890625, -0.681640625, 0.13037109375, 0.286376953125, 0.85205078125, 0.87548828125, 1.369140625, 2.03515625, -0.5986328125, -0.99462890625, -1.4052734375, 0.1842041015625, 0.8916015625, -0.6640625, 0.95654296875, -0.220458984375, 1.0146484375, -0.444091796875, -0.314453125, 1.583984375, -1.7919921875, 0.0024566650390625, 1.0625, -0.262939453125, 0.361083984375, -2.50390625, -0.301025390625, -1.208984375, -0.3583984375, -0.70654296875, -0.435546875, -0.419189453125, -0.7880859375, 0.98046875, -0.495361328125, -0.5849609375, 0.07159423828125, 0.419189453125, 1.3134765625, 0.8984375, -2.1484375, 0.11297607421875, 1.455078125, -1.2236328125, 0.491943359375, 0.11651611328125, -1.51171875, 2.03125, -0.6201171875, 1.04296875, -0.25341796875, 0.78662109375, -0.71923828125, -0.47265625, 1.6796875, -2.07421875, -0.2216796875, -1.107421875, -0.70654296875, 0.2301025390625, 0.84130859375, 0.4169921875, 0.68603515625, -0.5908203125, -0.64404296875, -1.04296875, -0.6005859375, 1.1865234375, 0.2900390625, -1.302734375, -1.533203125, -0.2225341796875, -2.10546875, -0.353515625, 0.92626953125, 1.03515625, -0.5791015625, 0.83935546875, -0.267333984375, 1.8349609375, -1.2900390625, -0.492919921875, 1.3115234375, 0.0333251953125, 0.56005859375, -0.8564453125, -1.7255859375, 0.71484375, 1.591796875, -0.2100830078125, -0.2164306640625, 0.8505859375, -1.1552734375, 0.87060546875, 0.591796875, -0.87548828125, -1.625, 1.6044921875, -0.281494140625, 0.87841796875, 1.7724609375, -0.673828125, 0.57861328125, -0.397216796875, 0.7998046875, -1.2548828125, 0.79638671875, 0.07843017578125, -0.410400390625, 0.203369140625, 1.3046875, 0.5732421875, 0.260009765625, -0.08819580078125, -2.150390625, 0.96484375, 0.3173828125, 1.1796875, -0.354248046875, 0.8916015625, 1.376953125, 0.260498046875, 0.385498046875, -1.947265625, -1.23046875, 1.3212890625, -1.765625, -0.73828125, 1.5224609375, 0.11212158203125, -0.798828125, -0.65283203125, 0.474609375, -0.21337890625, 2.25390625, -0.034759521484375, -0.97412109375, -0.126708984375, 0.226318359375, 1.1083984375, -0.6611328125, 1.0947265625, 0.63818359375, 1.0107421875, -0.83544921875, -0.1409912109375, 0.10198974609375, -0.8369140625, 0.0092315673828125, -0.65234375, 0.1448974609375, 0.29345703125, 0.39306640625, -0.892578125, -0.60107421875, -0.42822265625, 0.49462890625, -1.7236328125, 0.350830078125, 0.18603515625, 1.22265625, 0.87548828125, -0.6611328125, 0.5068359375, -2.720703125, -0.1158447265625, 0.5400390625, 1.6669921875, -1.126953125, -0.62890625, 0.057342529296875, 0.74853515625, 0.78076171875, -0.69970703125, -0.6240234375, 0.7685546875, 0.25048828125, 0.74072265625, -0.86962890625, 0.10614013671875, -0.396484375, 0.2421875, 1.7294921875, -0.22314453125, -0.98486328125, 0.1689453125, 0.29638671875, 0.42333984375, -0.10076904296875, 0.0638427734375, 1.1474609375, 0.63134765625, 1.08984375, 0.134033203125, -0.1949462890625, -0.611328125, -2.029296875, 0.2169189453125, -0.374267578125, -0.501953125, -1.0146484375, 1.197265625, 0.931640625, 0.8251953125, -0.06915283203125, -1.62109375, 0.1693115234375, 1.8779296875, -1.0517578125, -1.0576171875, 0.43603515625, 0.2186279296875, -0.06927490234375, -0.6357421875, -0.1654052734375, 0.309814453125, 1.580078125, 0.78466796875, -0.87890625, 1.2421875, 0.2086181640625, -0.31298828125, -1.357421875, 0.92724609375, 0.482421875, -1.1640625, 0.50732421875, 1.1923828125, 1.34375, -0.560546875, 1.3125, 0.10009765625, -0.173583984375, 0.1749267578125, 0.54833984375, 1.2841796875, 0.556640625, -0.91015625, 1.1767578125, -1.16796875, 2.7265625, -0.876953125, 0.433349609375, 0.0460205078125, -0.317626953125, -0.88232421875, -0.67919921875, -1.4384765625, -0.337890625, 1.1962890625, -0.34765625, 1.572265625, 0.50830078125, -0.01544189453125, 0.85498046875, -0.308349609375, -0.52783203125, -0.126220703125, 0.90625, 0.20068359375, 0.0268402099609375, 2.6953125, 1.154296875, 0.294921875, -0.15185546875, 1.2412109375, -0.028900146484375, 0.623046875, -2.08203125, -1.4736328125, -1.337890625, 0.78466796875, -1.8759765625, 1.0263671875, 0.2469482421875, 0.81689453125, 1.0283203125, 2.01171875, -0.6884765625, 0.363037109375, -0.26806640625, -0.197021484375, 0.166015625, -0.343017578125, 0.4541015625, 0.83740234375, -0.42578125, -1.4306640625, 0.8212890625, -1.0546875, 0.5654296875, -0.736328125, -0.54248046875, 1.6474609375, 0.29345703125, 1.26171875, 0.45654296875, 0.568359375, 0.07562255859375, -0.52880859375, -1.0439453125, -1.3544921875, -1.2919921875, 0.72705078125, -1.1796875, -0.0396728515625, 0.54296875, -0.458251953125, -0.293701171875, -0.529296875, 0.450927734375, 0.935546875, -1.3642578125, -0.888671875, 0.765625, -0.38525390625, -0.263427734375, -0.25390625, 0.54443359375, -0.26123046875, 0.239013671875, -0.9951171875, -0.63330078125, 0.3798828125, -0.27392578125, 1.091796875, -0.21142578125, -0.294677734375, -0.65771484375, -0.3173828125, 0.69189453125, 0.88134765625, -0.1168212890625, 0.92919921875, 0.77099609375, 0.95703125, -0.0970458984375, 0.95068359375, 0.347900390625, 0.57568359375, 0.5029296875, 0.62109375, -0.1917724609375, -0.0938720703125, 2.66796875, 0.270263671875, -2.103515625, -1.0166015625, -0.175537109375, -0.394287109375, 1.2236328125, -0.36376953125, 0.1756591796875, -2.0, -0.291748046875, 1.8125, 0.15771484375, -0.393310546875, -1.4287109375, 0.170654296875, 0.64892578125, 0.53369140625, 0.8662109375, -0.744140625, 0.352294921875, -0.35595703125, 0.0692138671875, -0.424072265625, 0.7880859375, 0.79931640625, 0.349609375, 0.72412109375, 0.37890625, -0.84130859375, 1.1318359375, 1.033203125, 0.49609375, -0.1453857421875, -0.6259765625, 1.2724609375, -0.44873046875, 1.6904296875, 1.1416015625, -0.5224609375, -0.96044921875, -0.962890625, -1.0791015625, 0.424072265625, 0.01446533203125, -0.473388671875, 0.2025146484375, -0.6005859375, 1.18359375, -2.33203125, -0.1494140625, -1.1650390625, -0.284912109375, 0.83154296875, -1.2890625, 0.038787841796875, 0.289794921875, -1.0595703125, 1.6142578125, 1.1904296875, -1.45703125, -0.82666015625, -1.6123046875, 2.017578125, -0.60009765625, 1.9345703125, 0.625, -0.53125, -0.7255859375, -0.9091796875, -0.99658203125, -2.15625, 1.2333984375, 1.140625, 0.2437744140625, 0.1243896484375, 1.703125, 0.0894775390625, 1.484375, 0.81982421875, -0.810546875, -0.004856109619140625, 0.99169921875, 0.2237548828125, 0.57373046875, 1.376953125, -1.576171875, -0.346435546875, -2.38671875, -0.8134765625, -0.1436767578125, -0.027557373046875, 0.1875, 1.115234375, 1.794921875, -0.29638671875, -2.03125, -0.0726318359375, -0.794921875, -1.0322265625, -1.365234375, 2.19921875, 0.93701171875, 0.1278076171875, -0.62158203125, 0.496826171875, -0.78857421875, 0.75634765625, -0.3056640625, 0.701171875, -0.703125, -0.384765625, -0.023223876953125, 1.0390625, 0.047393798828125, 1.2275390625, -0.418701171875, 0.6474609375, 2.09765625, 0.459716796875, -0.1846923828125, 0.413330078125, -0.01200103759765625, -0.2196044921875, -0.93359375, -0.07568359375, -0.156494140625, -0.1766357421875, 0.409912109375, 1.4775390625, -1.228515625, -0.00666046142578125, 0.345458984375, 0.32275390625, -0.9208984375, 0.80126953125, -1.6875, -1.3798828125, -0.337158203125, 0.70068359375, 0.139404296875, -0.364013671875, -0.2130126953125, -0.9892578125, -0.66455078125, 0.8974609375, -0.453369140625, 0.46435546875, -1.1396484375, 1.1884765625, -0.1170654296875, 1.0224609375, 0.22705078125, -0.6533203125, 1.2646484375, -0.7939453125, -1.3173828125, 2.71875, 2.8125, 0.198486328125, -1.111328125, 0.06597900390625, 0.60595703125, 0.5283203125, -1.03125, -0.371337890625, 1.3095703125, -0.0260009765625, 0.474853515625, 1.84765625, 2.703125, 2.494140625, 1.5595703125, 0.1163330078125, -2.33203125, -0.2177734375, 0.673828125, 0.397216796875, -1.2333984375, -0.96240234375, 0.0080413818359375, 0.388427734375, -0.3466796875, -0.1602783203125, -0.0789794921875, 0.5078125, 0.2705078125, -0.541015625, 0.58984375, -0.486083984375, -0.060516357421875, -1.408203125, 0.266357421875, 1.9951171875, 0.2467041015625, -0.4951171875, 0.7646484375, 0.58984375, -0.70166015625, 0.0308990478515625, 0.681640625, -1.6806640625, -0.56494140625, -0.400390625, 1.71875, 0.89892578125, 0.0892333984375, 0.42041015625, 0.22705078125, -2.19140625, -1.685546875, -0.1885986328125, 0.297119140625, -1.3447265625, 0.8037109375, -2.31640625, 0.08740234375, -0.412841796875, -0.20654296875, 0.78515625, 0.630859375, -0.85888671875, 0.1041259765625, 1.2841796875, -1.1787109375, -0.689453125, 1.0517578125, 1.525390625, -0.95361328125, 0.388427734375, -0.96142578125, 1.10546875, -0.43212890625, 0.52783203125, -0.7392578125, -1.93359375, -0.09259033203125, 1.6015625, 1.6669921875, -1.1435546875, -1.38671875, 0.93408203125, -0.5263671875, -0.435302734375, 0.290771484375, 0.669921875, -0.377685546875, 0.187744140625, 0.6455078125, 0.325439453125, -1.2265625, 0.5263671875, 1.765625, -0.01242828369140625, -0.57568359375, 0.84423828125, -0.427001953125, 0.452880859375, 1.203125, 0.366455078125, 1.171875, -1.1875, -0.276611328125, -1.658203125, 0.87841796875, 0.79150390625, 0.274658203125, -0.97900390625, -0.884765625, 0.33251953125, 0.3642578125, 0.496337890625, 0.494873046875, -0.2427978515625, -0.9287109375, 0.205078125, 1.650390625, -1.369140625, 1.345703125, 0.54296875, 1.16796875, 0.5283203125, 0.67041015625, -0.70751953125, -2.017578125, -0.10125732421875, -1.390625, 0.740234375, -0.50244140625, 0.0804443359375, 0.84423828125, -1.640625, -3.296875, -0.3603515625, -0.66064453125, 1.0908203125, 0.1734619140625, 0.050994873046875, 0.31201171875, 0.0022602081298828125, -0.198974609375, 1.078125, -0.25244140625, -0.1298828125, 0.611328125, 0.52783203125, -1.263671875, 2.40234375, 0.01214599609375, 0.27392578125, 1.2998046875, 0.385498046875, -1.279296875, -0.01387786865234375, -0.256591796875, 0.31103515625, -0.0015201568603515625, -1.0380859375, 0.82080078125, -1.0517578125, -0.3408203125, 0.2724609375, 1.18359375, -1.0654296875, -0.3818359375, 1.404296875, -0.20703125, -0.298828125, 0.33984375, -0.41845703125, -0.97216796875, -0.0948486328125, -0.320068359375, 0.39892578125, -0.4775390625, -1.4345703125, -0.2734375, 0.765625, -0.7548828125, -0.06268310546875, -1.552734375, 1.25390625, 0.08917236328125, -1.3505859375, 1.2119140625, 0.984375, 0.0240936279296875, 1.00390625, 0.144775390625, -1.1875, 1.01171875, -0.734375, -1.216796875, -0.474609375, 0.54638671875, 1.306640625, -0.7568359375, 1.19140625, -0.364990234375, -0.390869140625, 0.74365234375, 1.4248046875, 1.1318359375, -1.1884765625, 0.149658203125, -0.65771484375, 2.201171875, 0.36865234375, 0.4013671875, -0.1436767578125, -1.44921875, -0.1527099609375, -0.308837890625, 0.5439453125, -1.0, 0.22265625, 0.4990234375, -2.9296875, -0.50048828125, 1.267578125, -1.0576171875, -0.476318359375, 0.42333984375, -0.8720703125, 0.6591796875, 1.2880859375, 1.0498046875, -0.40673828125, 0.12469482421875, -1.7744140625, 0.3994140625, -0.658203125, 0.0672607421875, 0.1241455078125, 0.88330078125, -0.32958984375, 1.2587890625, -0.2042236328125, -0.6669921875, 0.086669921875, 3.240234375, 1.2646484375, -0.67529296875, -1.24609375, -0.020111083984375, -0.60498046875, -0.084716796875, -1.5361328125, -1.8125, 1.2275390625, -1.8173828125, 0.2451171875, 0.039703369140625, -1.6376953125, -1.1494140625, 0.7666015625, -0.6484375, 0.81494140625, -0.95166015625, 0.18798828125, -1.1875, 2.166015625, -1.2333984375, -0.47509765625, 0.46875, 0.258544921875, 0.156982421875, 0.97900390625, -1.2646484375, -0.53515625, -0.474853515625, -0.20166015625, -0.1842041015625, 1.0751953125, -1.650390625, -0.426513671875, -0.5673828125, 0.487060546875, 0.436767578125, -0.5380859375, 1.36328125, -0.051666259765625, 0.124267578125, 0.11181640625, -0.1878662109375, -1.369140625, 0.05499267578125, -0.703125, -0.66552734375, 0.095703125, -0.8662109375, -0.0928955078125, 1.3740234375, -0.9931640625, -1.3876953125, 0.177001953125, -0.1884765625, 0.304443359375, -0.63623046875, 0.62353515625, -0.87353515625, -0.43798828125, -1.0947265625, -0.3486328125, 0.5390625, -0.36767578125, -0.87158203125, 0.59375, 0.296142578125, -0.91796875, -0.91552734375, 0.74267578125, -0.334228515625, -0.40576171875, -0.943359375, -0.88525390625, -0.374755859375, -0.09747314453125, 1.7958984375, 0.890625, -0.1298828125, -1.1357421875, -0.353759765625, -0.1988525390625, 0.442626953125, -1.21484375, 0.7041015625, -0.8427734375, 0.689453125, -1.404296875, 0.8271484375, 0.372802734375, -0.876953125, 0.68505859375, -0.57763671875, -0.556640625, 1.060546875, -1.1708984375, 0.22509765625, -0.209716796875, -1.6337890625, -0.91796875, 1.876953125, -0.97265625, -0.701171875, -0.10986328125, 1.2529296875, -0.71875, -0.187744140625, 0.56787109375, 0.26953125, -1.380859375, -0.142333984375, -1.486328125, 0.89990234375, 2.236328125, -0.00604248046875, -0.95361328125, 0.0116729736328125, -0.56103515625, -0.3173828125, -0.208984375, -2.404296875, -0.09869384765625, -1.0634765625, -1.3515625, 1.69921875, -0.446533203125, -0.6748046875, 0.0823974609375, 0.292724609375, -2.375, 0.81689453125, 0.98779296875, 0.51318359375, -1.3505859375, 0.01203155517578125, 0.61474609375, 0.7080078125, -0.1763916015625, -0.393310546875, 1.171875, 0.86376953125, 0.126953125, -0.62890625, 1.6220703125, -0.53271484375, 0.052001953125, 0.0007796287536621094, -0.68798828125, 1.369140625, 1.068359375, 0.58203125, 0.91162109375, -0.1094970703125, 0.2890625, 0.2802734375, -0.99658203125, -0.003261566162109375, 0.7333984375, -0.250244140625, -0.403076171875, -0.69091796875, -0.29150390625, 0.1566162109375, 1.5009765625, -0.293212890625, 0.3779296875, -0.1822509765625, -0.0760498046875, -2.162109375, -0.0305938720703125, -0.02947998046875, -0.57373046875, -1.0537109375, -0.233642578125, -0.45166015625, -1.17578125, -0.20703125, -0.9677734375, -1.1689453125, -1.0869140625, -0.349365234375, -0.25439453125, 1.4736328125, 1.021484375, -2.25390625, 1.1279296875, -0.06585693359375, -1.64453125, -0.77294921875, -0.282470703125, 0.78515625, 0.814453125, 0.42724609375, -0.541015625, 0.6240234375, -0.1591796875, 2.705078125, 0.2734375, -0.418212890625, 0.10247802734375, -0.1947021484375, -0.978515625, 0.7607421875, -0.83984375, 0.411865234375, 0.058441162109375, 0.1505126953125, -2.435546875, -0.900390625, -1.37109375, 0.193115234375, 1.294921875, -0.273193359375, 0.1162109375, -0.56591796875, 0.29052734375, -1.626953125, -1.60546875, -0.333740234375, -0.047454833984375, -0.78515625, 1.283203125, 0.80810546875, 0.1817626953125, 1.04296875, -1.162109375, 1.1064453125, -0.1309814453125, 1.4267578125, -0.5654296875, -0.294189453125, 0.974609375, -0.81005859375, 0.1190185546875, 0.485107421875, 0.65185546875, -0.6826171875, 1.0107421875, -2.251953125, 1.4072265625, -0.96142578125, 0.274658203125, -0.313720703125, -0.1622314453125, 0.0479736328125, 0.1876220703125, 1.244140625, -0.91943359375, -1.3056640625, 0.060150146484375, 0.56005859375, -0.08441162109375, -0.401123046875, 0.50341796875, 0.30810546875, -1.01171875, 0.53759765625, -0.79150390625, -0.46875, 1.5595703125, -0.184326171875, -1.3447265625, 0.35693359375, -1.4912109375, 0.8994140625, 0.85888671875, 0.394287109375, 2.220703125, 0.324462890625, -1.9189453125, 0.9384765625, 1.1767578125, -0.13671875, 0.77587890625, -1.251953125, -3.005859375, -1.265625, 0.06939697265625, 0.05419921875, -1.884765625, -0.498291015625, 0.403076171875, 0.72412109375, -0.483642578125, -0.166259765625, 0.10980224609375, 1.4423828125, -0.387451171875, 0.380615234375, -0.2685546875, 0.90625, -0.319580078125, -0.9423828125, -0.7568359375, -1.2158203125, -0.2427978515625, -0.7216796875, 0.5078125, 0.2359619140625, -1.45703125, 1.599609375, 1.3251953125, -0.293212890625, 1.5908203125, -1.48046875, 0.99267578125, -0.44140625, 1.2490234375, 1.9248046875, -1.1181640625, 3.279296875, -1.9287109375, -1.244140625, -1.3359375, 1.8232421875, 0.67236328125, -0.197998046875, 0.9287109375, 1.0771484375, 0.26318359375, 0.580078125, 0.335693359375, -1.068359375, -1.3515625, -2.443359375, 0.779296875, 0.69970703125, -0.3447265625, -1.2470703125, -0.361083984375, 0.0751953125, 1.5361328125, -0.357421875, 1.1982421875, 1.2294921875, 1.7470703125, -0.595703125, -0.42724609375, 0.68212890625, -0.6640625, -0.306396484375, 0.3525390625, -1.37890625, -1.0966796875, -0.54150390625, -0.10015869140625, -1.3798828125, -0.56787109375, -0.77734375, -0.061004638671875, 0.0201873779296875, 1.3046875, 0.3125, 0.224365234375, 0.56103515625, 0.0394287109375, -0.77685546875, 0.9462890625, 0.53857421875, 0.08099365234375, -0.248779296875, -0.615234375, 0.685546875, 0.94384765625, -0.67724609375, -0.908203125, -0.62255859375, -1.44140625, 1.50390625, 1.24609375, -0.50927734375, 0.560546875, -0.267822265625, 0.7490234375, -0.662109375, 0.2491455078125, 0.1195068359375, -1.828125, -0.5146484375, -0.67041015625, 0.467041015625, -0.74560546875, 1.244140625, 0.418212890625, 0.5615234375, 0.263671875, -1.38671875, 0.7080078125, -0.2919921875, 1.2216796875, 1.4443359375, -1.5966796875, 0.049713134765625, 2.6640625, 0.1396484375, 1.4873046875, 0.424560546875, 0.346435546875, -0.95263671875, -0.469970703125, 0.1480712890625, 0.56591796875, -0.80908203125, -0.324462890625, -1.91015625, -2.2421875, 1.1240234375, 0.72900390625, -1.3984375, -1.068359375, 0.260009765625, -0.45849609375, 0.90478515625, 1.48828125, -0.1409912109375, -1.1376953125, -1.30078125, -1.53125, 0.162841796875, 0.321044921875, -0.6142578125, -0.431640625, -1.8046875, 1.37890625, 0.375, -0.4501953125, 1.0048828125, -0.4072265625, 2.130859375, 0.428466796875, -0.5986328125, 3.173828125, 0.2119140625, -0.60546875, -0.5576171875, -1.1474609375, -0.0869140625, 0.057861328125, 2.388671875, -0.8251953125, 0.437744140625, -0.025726318359375, -0.045654296875, 1.0986328125, -1.029296875, -0.630859375, 1.0869140625, 0.200927734375, 0.0653076171875, 0.1904296875, -0.6796875, -0.033355712890625, -0.8662109375, 1.6728515625, 0.7373046875, -0.091552734375, -0.14404296875, 1.0654296875, -0.10540771484375, 0.2288818359375, 0.392333984375, 1.4619140625, 2.203125, -2.56640625, 0.74853515625, -0.4033203125, 0.10260009765625, -0.3525390625, 0.1629638671875, -0.64892578125, 0.1739501953125, -0.09283447265625, 0.312744140625, 0.966796875, 2.259765625, 0.04644775390625, -2.24609375, -0.35009765625, 0.984375, -0.759765625, -0.8408203125, -0.2066650390625, -0.402099609375, 1.787109375, 1.5595703125, -0.97705078125, 0.60107421875, 0.08209228515625, -0.31689453125, 2.720703125, 1.767578125, -1.6171875, 0.16259765625, 0.64404296875, 0.1630859375, 0.61279296875, -0.266845703125, 0.2447509765625, 0.9091796875, -1.2607421875, 1.103515625, 0.354248046875, -0.19775390625, -2.099609375, 1.1884765625, 0.1719970703125, 0.794921875, 0.44189453125, -0.76416015625, -1.609375, 1.232421875, 0.2388916015625, 0.76806640625, 1.2265625, -2.603515625, -0.974609375, 0.87353515625, -0.2496337890625, -1.1494140625, 0.4052734375, -0.07073974609375, -1.2265625, 0.96240234375, 1.859375, -3.1875, 2.361328125, -0.7705078125, -0.340576171875, 1.1181640625, 0.544921875, 0.5, 1.56640625, -0.67578125, 0.08697509765625, 1.3642578125, -0.75390625, 0.4599609375, 0.4111328125, 0.278564453125, 0.6904296875, 0.304931640625, 0.1297607421875, 0.108154296875, -0.86279296875, -0.0289764404296875, 0.2451171875, 0.52587890625, -0.350341796875, -0.396240234375, -0.455078125, 1.1171875, -0.439208984375, 0.8388671875, 0.04998779296875, -1.419921875, -0.338623046875, -1.1591796875, 0.9521484375, 0.52978515625, 0.259521484375, 0.32470703125, 0.451904296875, 0.23779296875, 0.50830078125, -1.271484375, 1.56640625, -2.548828125, 0.62255859375, -0.51318359375, -1.5224609375, 0.33544921875, -1.966796875, 1.0615234375, 1.8310546875, -0.327392578125, -0.45263671875, 2.11328125, 0.2442626953125, -0.6494140625, 0.046478271484375, 1.4345703125, -1.7607421875, -0.673828125, 0.07977294921875, -1.4580078125, 0.4560546875, 0.52734375, 0.75244140625, 0.1773681640625, 1.2646484375, -2.275390625, -0.7783203125, 0.87158203125, -1.064453125, -0.374755859375, -2.001953125, -1.4560546875, -0.24658203125, -0.54150390625, -0.580078125, 1.419921875, -2.361328125, 1.1201171875, 1.166015625, -0.48291015625, -0.26123046875, -0.830078125, -0.71484375, 1.5556640625, -1.0859375, -1.2685546875, -2.607421875, 0.024139404296875, -0.64111328125, -0.3369140625, 0.50146484375, -0.275390625, -1.546875, -0.339111328125, -0.2607421875, 0.39453125, -0.1875, 0.74853515625, -1.4296875, 1.0419921875, 0.640625, -2.26171875, 0.224609375, -0.93505859375, -0.98876953125, 0.767578125, 1.0751953125, -0.450439453125, -0.0745849609375, -0.65966796875, -0.55908203125, 0.195068359375, -1.4462890625, 0.266357421875, 1.0244140625, 1.287109375, -1.1083984375, 1.501953125, 1.52734375, -0.290771484375, 0.37060546875, -1.421875, 0.69140625, 2.1484375, 1.2392578125, -1.2060546875, 0.57373046875, 2.185546875, 0.7578125, 0.392822265625, 2.265625, -0.20947265625, 2.1953125, 0.4404296875, -0.4892578125, -0.212158203125, 0.52734375, 0.994140625, 0.33349609375, 0.1378173828125, -1.2744140625, 0.11846923828125, 0.09429931640625, -0.69873046875, 0.51708984375, -1.2138671875, 3.16796875, -0.986328125, -1.2783203125, 0.1385498046875, -0.398681640625, 1.1533203125, 0.7294921875, -2.19921875, 1.0439453125, -0.61669921875, 0.2374267578125, 0.302734375, -1.3466796875, -0.357177734375, 0.271240234375, 0.1456298828125, -0.2110595703125, -0.16943359375, -1.6923828125, 0.2362060546875, -0.1959228515625, 0.2734375, 0.5732421875, -1.7041015625, -0.1094970703125, -0.93994140625, -2.17578125, -1.0419921875, -0.96484375, 1.26171875, -1.0439453125, 0.47998046875, -1.2158203125, -0.04425048828125, -0.685546875, -0.1802978515625, 1.826171875, -3.01953125, 0.2474365234375, 0.15576171875, 1.4091796875, -0.53125, 0.6357421875, -1.32421875, -0.03656005859375, -0.0921630859375, -0.80078125, -0.37548828125, -0.1885986328125, 0.432373046875, 0.6875, 0.68701171875, 0.935546875, -0.94970703125, -0.0421142578125, 0.279541015625, -0.2198486328125, 0.0865478515625, -0.486328125, 0.9755859375, -1.162109375, 1.775390625, 0.332275390625, 1.607421875, 1.251953125, -1.2490234375, -1.568359375, -0.279296875, 0.131591796875, 1.26953125, -0.0338134765625, -0.8701171875, -2.513671875, 0.0303192138671875, -1.7353515625, 0.8779296875, 1.146484375, -1.0810546875, -0.26123046875, 0.51416015625, -0.322509765625, 0.70751953125, -1.6455078125, -0.54541015625, -0.3818359375, -1.111328125, 1.9765625, -0.1878662109375, -0.284912109375, -0.375244140625, 0.32373046875, 1.6708984375, 1.13671875, -0.626953125, 0.49462890625, -1.5576171875, 0.30517578125, 0.03753662109375, -0.5810546875, -0.80712890625, 0.00464630126953125, 1.578125, 0.39404296875, 1.1796875, 0.367431640625, 1.0, 0.474609375, -0.175048828125, 2.494140625, -0.87353515625, 0.66357421875, -1.2275390625, 0.73876953125, 1.20703125, -1.451171875, -0.29150390625, 0.106689453125, -0.341796875, -0.65771484375, -0.72998046875, -0.01367950439453125, -1.2705078125, 0.7744140625, -0.5458984375, -0.297119140625, -0.69384765625, 1.26953125, -2.724609375, -0.96484375, 0.951171875, 0.787109375, 1.3623046875, 0.236328125, -0.07452392578125, 0.194580078125, 1.6875, -0.54736328125, 0.0582275390625, -2.76171875, -0.250244140625, 0.1683349609375, 0.66796875, -1.0712890625, 0.701171875, -1.2216796875, -0.485107421875, -0.90966796875, -0.2325439453125, -0.41650390625, 1.5263671875, 0.496826171875, -0.5234375, -0.556640625, -1.2734375, -1.51171875, -0.25732421875, 1.0478515625, 1.287109375, -0.2110595703125, -0.33740234375, 1.2451171875, 0.50244140625, 0.2332763671875, 0.464599609375, 1.115234375, -0.1353759765625, -0.8154296875, -0.475830078125, -0.089599609375, 2.515625, 0.337890625, -1.015625, 0.495361328125, -0.0751953125, -0.7177734375, 0.41064453125, 0.2291259765625, 1.5712890625, 0.89453125, 0.27294921875, 0.9423828125, -0.1416015625, 0.429931640625, -0.50341796875, -0.192626953125, -1.810546875, -0.9228515625, -1.1943359375, -0.869140625, 0.231201171875, 0.2705078125, 0.99560546875, 0.085693359375, -1.9150390625, -0.36572265625, -1.2861328125, -0.09869384765625, 1.3427734375, -0.445556640625, -0.1015625, -1.7177734375, 0.7861328125, -2.072265625, -0.68994140625, 0.168212890625, 0.92919921875, 0.00965118408203125, 0.496337890625, 0.49072265625, 0.5791015625, 0.2418212890625, 1.9287109375, -0.568359375, -0.78076171875, -1.083984375, -0.7080078125, 0.414306640625, 0.391845703125, -0.53125, -0.58203125, -0.7587890625, 0.290283203125, 0.70068359375, -2.365234375, -0.477294921875, 0.669921875, 1.2265625, -0.69873046875, 0.62060546875, 1.2138671875, 1.1240234375, 0.79638671875, 0.4384765625, 0.6337890625, -1.4951171875, -1.6533203125, 0.74853515625, 0.0262603759765625, -0.51904296875, 1.9765625, -0.92431640625, -0.896484375, -0.23974609375, 0.86767578125, -1.4716796875, -1.138671875, -1.0654296875, 0.066162109375, 0.29736328125, -1.560546875, 0.1658935546875, -0.9501953125, 0.39794921875, -0.568359375, 0.0094451904296875, -0.8857421875, 1.4951171875, -0.413818359375, 0.3564453125, 1.1650390625, 0.1783447265625, 0.3505859375, -0.99072265625, 0.06439208984375, 0.1842041015625, 0.154541015625, 0.028472900390625, -1.5107421875, -0.10308837890625, 0.252685546875, -2.216796875, 0.2335205078125, 1.833984375, 2.90234375, 1.9111328125, 0.748046875, -0.82080078125, -1.0458984375, -1.5390625, -1.9560546875, -1.0205078125, 0.27294921875, -0.50732421875, -0.77978515625, 0.65380859375, -0.2161865234375, 0.2315673828125, -0.66455078125, -0.8349609375, 0.63330078125, -0.9091796875, -1.1572265625, -0.3681640625, 0.70068359375, -0.63037109375, -0.01032257080078125, 0.0085296630859375, 3.40234375, 1.2060546875, -0.80078125, 0.4013671875, 0.60302734375, 1.7978515625, -1.1708984375, 0.11956787109375, 0.324951171875, -1.9716796875, 0.0249481201171875, -0.151123046875, 2.009765625, 1.1201171875, -0.30322265625, -2.279296875, -0.12396240234375, -0.30908203125, 1.1611328125, -1.5791015625, 1.0625, 0.135986328125, -1.373046875, -0.35693359375, 0.65673828125, 0.006084442138671875, 2.02734375, 1.9794921875, -0.57958984375, -0.44091796875, -1.576171875, 0.5966796875, -0.5947265625, 1.083984375, -0.6064453125, 2.00390625, -0.372802734375, -0.67041015625, -0.29736328125, 0.8486328125, 0.9267578125, 0.137939453125, -0.89208984375, 0.43310546875, -1.12890625, 0.92138671875, -0.499755859375, -0.3173828125, 0.6416015625, -0.615234375, -0.0014772415161132812, 0.9384765625, -0.86669921875, -1.1865234375, 1.421875, 0.442626953125, -0.1473388671875, 1.1396484375, 0.07037353515625, -1.3173828125, -0.82861328125, -0.1575927734375, -0.4521484375, -0.250732421875, -0.953125, -0.03753662109375, -1.62890625, -0.57958984375, 0.2486572265625, 1.486328125, 1.2470703125, -0.00833892822265625, -1.4150390625, 0.29150390625, -0.30322265625, -0.85595703125, -0.2242431640625, 0.5830078125, -1.6484375, 0.48876953125, 0.1561279296875, 1.2197265625, -0.53173828125, 0.73193359375, 1.4873046875, -2.32421875, -1.736328125, -1.947265625, -0.736328125, -0.60498046875, -0.2269287109375, 0.39013671875, -0.1396484375, 1.5380859375, -2.52734375, 1.208984375, -0.497802734375, 1.0869140625, -0.05072021484375, 0.93408203125, -0.99169921875, 0.521484375, 2.078125, 2.744140625, 0.93408203125, 1.265625, -0.64599609375, 0.0265960693359375, 0.74072265625, 0.99755859375, -0.7109375, 1.986328125, -1.0576171875, -0.509765625, 1.0, -0.3701171875, -1.66015625, -0.0498046875, -1.0224609375, -1.166015625, 0.09161376953125, -0.71875, -0.3623046875, 0.02874755859375, 0.07379150390625, 0.440673828125, -1.013671875, 0.55517578125, 1.798828125, 0.0230865478515625, -0.0648193359375, 1.583984375, 0.12078857421875, 0.2210693359375, 1.0078125, -0.264892578125, -1.916015625, 1.1083984375, 1.0927734375, -1.3515625, -0.25830078125, -0.654296875, 0.2509765625, -0.63916015625, -0.95166015625, 0.66455078125, 0.61767578125, 1.2275390625, -0.9990234375, 0.69775390625, -0.09722900390625, 1.158203125, 0.0673828125, 0.5244140625, 0.037384033203125, 0.487548828125, -0.63232421875, -3.263671875, 1.2783203125, -0.258056640625, -0.3125, -0.69873046875, -0.6845703125, -0.01041412353515625, -0.407958984375, 0.89501953125, -0.451171875, -1.171875, 0.517578125, 0.487060546875, -0.0158538818359375, -0.1466064453125, -2.0703125, 0.244873046875, 1.1337890625, 0.8759765625, -0.2442626953125, -0.30810546875, 0.85400390625, 0.48779296875, 0.6923828125, -0.5419921875, -0.051361083984375, 0.0267486572265625, 1.3193359375, 0.79296875, -0.030426025390625, -0.404541015625, 0.0226287841796875, -0.212646484375, 0.0328369140625, -0.06463623046875, 1.501953125, 0.3876953125, 0.149169921875, -0.0662841796875, 0.60595703125, 0.498046875, 1.3583984375, 0.031829833984375, 0.93896484375, -1.1708984375, -1.248046875, -0.7841796875, -1.1640625, 0.010467529296875, 0.5830078125, -0.01922607421875, -0.30078125, 0.394287109375, -0.11114501953125, -0.11480712890625, 0.05255126953125, -0.4716796875, 0.1409912109375, -0.59326171875, 0.7568359375, 1.5615234375, -1.7431640625, 1.5859375, -0.3115234375, 1.58984375, -0.0869140625, -1.08984375, -0.0416259765625, -0.4345703125, -0.857421875, -0.30029296875, 0.55029296875, -0.468505859375, 1.2138671875, 1.3740234375, 0.044342041015625, 1.34375, -0.07720947265625, -0.35986328125, 0.375244140625, 0.72705078125, -0.2578125, -0.277587890625, -1.048828125, 0.046844482421875, 0.7919921875, 2.017578125, 2.310546875, -2.10546875, 0.050872802734375, 0.0469970703125, 0.578125, 0.08056640625, 1.1328125, 0.6865234375, 0.75, -0.31396484375, 1.009765625, -0.030242919921875, 0.212158203125, 0.64306640625, -0.6875, 0.41552734375, -0.61328125, -1.17578125, 0.036529541015625, -0.52490234375, 0.21728515625, 0.62841796875, -2.02734375, -0.17333984375, -0.363037109375, -0.464599609375, 0.42041015625, -0.552734375, 0.2491455078125, 1.416015625, -2.2890625, -0.4931640625, 0.79736328125, -0.57666015625, 1.5341796875, 0.1409912109375, 1.634765625, 0.85107421875, -1.5048828125, 0.159423828125, -0.33544921875, -0.92138671875, -0.1383056640625, -1.4638671875, 0.1663818359375, -1.4609375, 0.46435546875, -0.53173828125, -0.19873046875, -0.1781005859375, 1.2529296875, -2.259765625, 0.30029296875, -0.6826171875, -2.0703125, -0.74755859375, 1.5068359375, 0.96044921875, 0.07763671875, -2.162109375, 0.67578125, -0.369873046875, -0.5439453125, -0.357666015625, 0.2099609375, 0.327880859375, 0.84521484375, 1.2333984375, -0.6396484375, 0.67822265625, 1.6015625, -0.025054931640625, -1.1357421875, -3.1484375, 0.238037109375, 1.126953125, 1.0126953125, 1.888671875, -1.5810546875, -0.08843994140625, -0.79248046875, 2.623046875, 0.7998046875, 0.58935546875, -0.163330078125, 1.115234375, -0.54443359375, 0.01326751708984375, 0.81005859375, 0.19775390625, 0.71826171875, 0.2294921875, 0.390380859375, 0.042572021484375, -0.1539306640625, 0.2880859375, -0.030364990234375, -0.52490234375, 0.7353515625, 0.32958984375, 0.0994873046875, 1.654296875, 0.5185546875, -0.15966796875, 0.1343994140625, 0.79296875, -0.572265625, -2.12890625, -1.9951171875, -0.7509765625, 0.82421875, 0.11376953125, 0.37109375, -0.91064453125, 1.1396484375, -1.0966796875, -0.056549072265625, 0.38623046875, -0.74755859375, -0.93115234375, -0.91650390625, 0.8505859375, -0.5517578125, -0.409423828125, -1.5302734375, -0.56494140625, -1.2841796875, -1.0458984375, 1.275390625, 0.6796875, 1.4951171875, -0.293212890625, -1.6591796875, 0.74658203125, 0.58837890625, 0.82763671875, 0.6611328125, -1.498046875, -1.1865234375, -0.1805419921875, 0.28466796875, -0.6298828125, 0.712890625, 0.2064208984375, 0.3056640625, 0.460693359375, 1.642578125, -2.546875, 1.765625, -0.410400390625, 0.7880859375, 0.10369873046875, 0.7939453125, 1.09375, -0.765625, 0.7998046875, 0.07208251953125, 0.72021484375, -0.99658203125, 0.412353515625, 0.62890625, 0.83349609375, -0.05938720703125, 1.513671875, 0.1966552734375, -0.48876953125, 0.3310546875, 0.2259521484375, -0.1524658203125, 0.04620361328125, -0.1741943359375, -0.61083984375, 0.16552734375, -0.71435546875, -0.033111572265625, 1.1484375, 0.857421875, -1.03515625, 2.1796875, 0.428466796875, -0.552734375, 1.2431640625, 0.179443359375, -1.5009765625, -0.6884765625, -0.012054443359375, 1.0751953125, -0.20654296875, -0.037353515625, -0.98828125, 3.19921875, 0.174072265625, -0.99072265625, 0.290771484375, -0.39794921875, -0.58203125, -0.697265625, -0.69287109375, -0.427978515625, 0.77783203125, 0.15478515625, 0.564453125, 0.5302734375, 0.84619140625, 0.9345703125, -0.0187530517578125, -1.1357421875, -3.251953125, 1.87109375, 0.34326171875, -0.63818359375, 2.212890625, 0.1190185546875, -0.5419921875, 0.1217041015625, -2.98046875, 0.029754638671875, 0.5615234375, 0.38427734375, 1.1728515625, -0.39794921875, -0.9970703125, 0.56103515625, -1.6494140625, -1.1220703125, 1.2529296875, 0.494140625, -1.046875, -0.47509765625, -0.288818359375, -0.1658935546875, -0.03436279296875, 0.475341796875, -0.457275390625, 1.052734375, -0.0926513671875, -0.7734375, 1.26171875, -0.97216796875, 1.2353515625, 2.423828125, 0.0780029296875, 0.83349609375, -3.3984375, -0.24365234375, 1.2353515625, 0.46337890625, 0.583984375, -0.54052734375, 0.53173828125, -0.460693359375, 0.07470703125, -0.70263671875, 0.779296875, 0.890625, 2.12109375, 1.939453125, 0.71044921875, -0.045623779296875, 0.6318359375, -0.4306640625, 0.85205078125, -0.16162109375, -0.78857421875, 0.26416015625, -2.078125, 0.95166015625, 0.469482421875, 0.11187744140625, 1.109375, -0.91650390625, 0.99462890625, -0.7294921875, 0.51904296875, -2.328125, 0.28076171875, -0.461181640625, 0.32666015625, 1.5595703125, -0.487548828125, -0.61181640625, -0.775390625, -0.431396484375, -0.85498046875, -0.71435546875, -0.1263427734375, 0.7421875, 0.080078125, -0.2425537109375, -1.0205078125, 0.87744140625, 1.005859375, -0.09991455078125, -0.5087890625, 0.63720703125, 0.7294921875, 0.497802734375, -0.77685546875, 1.2314453125, 0.2890625, 0.4296875, -1.6806640625, 0.9560546875, 0.5185546875, 1.1552734375, 1.419921875, 0.73828125, 0.978515625, 0.219482421875, -1.2080078125, 0.78369140625, 1.0595703125, -0.5830078125, 1.0654296875, 0.142333984375, -1.0537109375, -1.9892578125, -0.038726806640625, -0.9404296875, 1.0771484375, -1.076171875, 0.405517578125, -0.202392578125, 1.1396484375, -0.10797119140625, 0.52734375, -0.99853515625, -0.42626953125, -0.341796875, 0.78662109375, -1.5244140625, -0.4365234375, 0.7529296875, -0.0611572265625, -0.9765625, -1.4208984375, 2.966796875, -1.87890625, -0.10882568359375, -0.9189453125, -1.322265625, -0.99755859375, -1.5556640625, 0.478515625, -1.5205078125, 0.462890625, 0.34619140625, 0.25048828125, 0.50732421875, 0.53271484375, -0.68505859375, 0.673828125, 0.2130126953125, 0.1199951171875, 0.1239013671875, 2.146484375, -0.697265625, -0.1700439453125, -0.288330078125, 1.6044921875, -0.845703125, -0.70068359375, 1.27734375, -0.357666015625, -1.9248046875, 0.433349609375, 1.248046875, 1.0537109375, -0.41357421875, 2.27734375, 1.8505859375, 2.8203125, 0.1612548828125, 0.7255859375, -0.1468505859375, 0.2861328125, -0.85546875, -0.291259765625, -0.2724609375, -0.1431884765625, 0.1932373046875, 1.0478515625, -1.2421875, 1.021484375, 0.495361328125, 1.046875, -0.1419677734375, 0.82861328125, 0.8857421875, -0.08697509765625, -0.448974609375, 0.72607421875, -0.448974609375, -0.98095703125, 0.53466796875, 0.41162109375, -1.6767578125, -0.60302734375, -0.9677734375, 0.439697265625, 0.69482421875, -1.021484375, -1.6435546875, -1.5595703125, -0.319580078125, -1.783203125, -0.13623046875, -0.66748046875, -0.28662109375, 0.556640625, 0.92431640625, 0.084228515625, 0.2308349609375, 0.85546875, 1.3203125, -0.58984375, -0.02618408203125, -0.7724609375, -0.056488037109375, -1.7236328125, -0.75927734375, -0.379638671875, -0.431396484375, -1.017578125, 0.381591796875, -1.607421875, -0.57666015625, 0.03692626953125, -1.26171875, -2.6875, -2.51171875, -2.193359375, 1.337890625, 1.388671875, -1.7802734375, -0.139404296875, 1.0068359375, -1.2314453125, -0.1978759765625, 1.8994140625, 1.216796875, -0.9921875, 0.42626953125, 1.3623046875, 1.267578125, 0.435791015625, -0.33154296875, -0.7236328125, 0.146728515625, 0.8681640625, -0.42578125, -0.2646484375, -0.1798095703125, -0.7060546875, -0.369140625, 1.99609375, -1.677734375, -0.30224609375, 1.1259765625, -1.244140625, 2.283203125, 1.560546875, 1.1064453125, -1.947265625, 0.475341796875, 0.2125244140625, 1.7861328125, 0.86181640625, -0.347900390625, 0.5439453125, 2.0859375, 0.28955078125, 0.0208587646484375, -0.02166748046875, -0.477783203125, 1.66015625, -2.14453125, 3.75, -1.630859375, 2.27734375, 0.472412109375, -0.80224609375, 0.425048828125, -0.7998046875, -0.62109375, -0.87646484375, 0.46923828125, -2.36328125, -0.00392913818359375, 0.51904296875, 1.0078125, 0.238037109375, -0.42138671875, -0.08453369140625, -1.2880859375, -2.5078125, -0.7568359375, -0.08404541015625, -0.32470703125, 1.6533203125, -0.1348876953125, 1.2431640625, 1.056640625, 0.4853515625, 0.5478515625, -1.623046875, 1.11328125, -0.1968994140625, 1.162109375, -1.2197265625, -0.8701171875, -0.3095703125, 1.19921875, 0.42041015625, -1.552734375, -1.1748046875, -0.2435302734375, -0.403564453125, -0.80908203125, 1.8994140625, 1.0966796875, -0.317626953125, 0.50537109375, 0.23828125, 0.927734375, 1.8818359375, 1.4189453125, 1.435546875, -0.385009765625, -0.2158203125, -1.0615234375, 0.08660888671875, -0.43505859375, -0.51708984375, -1.2060546875, -2.779296875, 0.38916015625, -0.96240234375, 0.07928466796875, 0.806640625, -0.56201171875, 0.12445068359375, -1.97265625, 2.0078125, -0.2978515625, 1.0078125, -0.3544921875, -0.5341796875, 0.00385284423828125, -0.44580078125, -1.255859375, -0.98974609375, 0.344482421875, -0.98046875, -0.0180206298828125, 1.296875, -1.1318359375, -0.435546875, -1.744140625, -0.1414794921875, 0.8955078125, -1.611328125, -0.57958984375, 0.31396484375, 0.476806640625, -1.837890625, 0.5556640625, -0.88818359375, 0.8525390625, -2.013671875, -0.2496337890625, -1.7412109375, -0.6826171875, -0.80859375, 1.7958984375, -0.222900390625, 3.22265625, -0.1580810546875, -0.1390380859375, 2.1015625, 0.58740234375, -0.25439453125, 0.220947265625, -0.4287109375, -1.1787109375, 0.79248046875, 1.021484375, -1.896484375, 0.63427734375, 0.0970458984375, 0.96044921875, 1.271484375, -0.57421875, -0.152099609375, -0.020416259765625, -0.453125, 0.06561279296875, -1.2265625, -0.6103515625, -0.388671875, -0.490478515625, -0.60302734375, -0.30322265625, -0.5263671875, -0.076904296875, 0.5751953125, -1.0439453125, 0.908203125, -2.197265625, -1.0791015625, -1.3408203125, 0.2802734375, 0.5712890625, -0.193115234375, -0.023101806640625, 1.8349609375, 1.07421875, 1.3369140625, 0.46533203125, -0.481689453125, 0.304443359375, 0.1785888671875, -0.447265625, -1.134765625, -0.54150390625, -0.0272064208984375, -0.023406982421875, -0.1715087890625, -0.1876220703125, 0.462646484375, -0.1590576171875, 0.75, 0.2093505859375, -0.18994140625, -0.53662109375, 2.28515625, -0.42919921875, -0.340576171875, 0.51708984375, 0.61376953125, 1.2998046875, -0.29638671875, -1.4609375, -0.31201171875, -1.025390625, -0.225341796875, 0.734375, 1.2099609375, 0.7451171875, 0.479248046875, -1.0380859375, -3.833984375, 0.55078125, 0.80908203125, 0.14111328125, 0.231201171875, -1.326171875, -0.177978515625, -1.6689453125, 0.7197265625, -0.4052734375, -0.525390625, -0.379150390625, 0.85546875, 0.128173828125, 0.255126953125, -0.0167694091796875, -1.2626953125, -0.40087890625, -2.03125, -1.673828125, -0.1368408203125, -0.159912109375, -0.486328125, -0.79736328125, 0.2174072265625, 0.5263671875, -0.1138916015625, 1.4765625, 0.75244140625, 0.3720703125, -0.5390625, -0.1932373046875, -0.2255859375, 1.3369140625, -0.73876953125, -0.4853515625, -0.455078125, 1.5078125, -0.255126953125, -0.7119140625, -1.390625, -0.767578125, 0.53955078125, -0.06005859375, 0.51611328125, -1.048828125, -0.1300048828125, -0.489501953125, -1.0771484375, 0.3203125, -0.41455078125, -0.92333984375, -0.5595703125, -0.52587890625, -0.283935546875, 0.8818359375, -1.3349609375, 0.2476806640625, -0.1031494140625, -0.29541015625, -0.6318359375, 1.4091796875, 2.728515625, 0.1676025390625, 3.01953125, 0.2041015625, -2.435546875, 0.490966796875, 1.5263671875, -0.439453125, -0.6240234375, 0.44384765625, -0.480712890625, -0.62353515625, -0.8427734375, 0.90087890625, -1.3671875, -1.6513671875, 0.86767578125, -1.2578125, 2.341796875, 1.630859375, -0.45556640625, 0.07666015625, 0.5869140625, 0.032257080078125, -0.44677734375, -0.0255126953125, -0.355224609375, 0.06878662109375, -0.2261962890625, 0.3984375, 0.391357421875, 0.98681640625, -0.485107421875, -0.40380859375, 0.90380859375, -0.313720703125, -1.841796875, -0.353515625, -0.08892822265625, -1.453125, 0.3095703125, -0.08148193359375, 0.09906005859375, 0.08453369140625, -0.88916015625, 1.314453125, 0.82080078125, -0.439453125, -0.1336669921875, 1.4111328125, -0.471435546875, -0.4296875, 0.1429443359375, 1.875, 1.51171875, -0.1265869140625, 0.09051513671875, 0.10125732421875, 1.1123046875, 0.1807861328125, 0.5673828125, 0.46533203125, 1.263671875, 0.1505126953125, -1.1279296875, -0.94873046875, 0.251708984375, 0.268310546875, -0.91796875, -1.6611328125, 0.2169189453125, -0.060943603515625, 0.8525390625, 0.55078125, -1.48046875, 0.96826171875, -0.2320556640625, -0.348388671875, -0.304931640625, 1.98046875, -1.0361328125, 0.640625, 1.453125, -0.387939453125, 1.0478515625, 0.83544921875, -0.451171875, 0.97119140625, -1.04296875, -0.91015625, -0.3486328125, -0.70068359375, -1.8447265625, 1.05078125, 0.775390625, -0.1298828125, 0.7890625, -0.463134765625, 0.495361328125, -2.13671875, 0.330078125, 1.3759765625, 0.390869140625, -1.107421875, -0.2401123046875, -0.51513671875, -0.6025390625, 0.08978271484375, -0.53515625, -1.75, 0.72705078125, 1.1513671875, 0.314208984375, -0.237060546875, -0.2471923828125, -0.52783203125, 1.0888671875, -0.041259765625, 0.49072265625, 1.1982421875, -1.3232421875, -0.05499267578125, -0.77734375, -0.191162109375, -1.5224609375, -0.97705078125, -0.317626953125, -1.857421875, -0.04180908203125, 0.2396240234375, 0.94921875, 1.8408203125, -1.0625, -0.56005859375, -0.580078125, 0.44482421875, -0.6220703125, -0.916015625, 0.7734375, 0.92138671875, -0.191650390625, -1.2607421875, 1.82421875, 1.732421875, -0.335693359375, -0.60107421875, -0.53466796875, -0.736328125, 0.490234375, 0.32666015625, 0.036712646484375, 1.0947265625, 0.986328125, -0.91259765625, 0.26611328125, -0.96630859375, -0.6787109375, 1.4462890625, -0.1932373046875, -1.64453125, 0.296142578125, -0.227783203125, 0.51513671875, 1.939453125, 0.859375, -0.08453369140625, 0.05712890625, 0.12371826171875, -1.6416015625, -0.036590576171875, -0.8212890625, 0.433837890625, 0.44921875, 1.9970703125, 0.1549072265625, 1.845703125, -0.003444671630859375, -0.34619140625, -0.48046875, 0.2440185546875, 0.720703125, -1.7529296875, -0.54638671875, -0.061920166015625, -0.318115234375, 0.4111328125, 0.394775390625, 1.0927734375, 0.33447265625, 0.92724609375, -0.400390625, -0.109375, 1.1884765625, -0.64892578125, -1.33984375, 1.8232421875, -1.4501953125, -1.4267578125, -1.1923828125, -0.7626953125, -1.21484375, -0.82861328125, 0.00745391845703125, 1.607421875, -0.61767578125, 0.12420654296875, -0.11199951171875, 0.345703125, -0.5, 0.6357421875, 0.50634765625, 3.2265625, 2.71484375, 0.349853515625, 1.6015625, 0.412109375, -0.369873046875, -1.6162109375, 1.1826171875, -1.546875, -1.296875, 1.03515625, -0.2447509765625, 0.5126953125, -0.65576171875, 0.409912109375, -0.8203125, -1.3095703125, -0.15966796875, 0.27783203125, 0.5791015625, 0.492919921875, -0.47119140625, -0.05206298828125, 2.220703125, -1.673828125, 0.6572265625, -1.234375, -0.888671875, -0.305419921875, -0.72705078125, 0.70263671875, -0.666015625, 0.80859375, 0.36474609375, 0.7529296875, 0.35546875, 0.1146240234375, 1.591796875, 1.4658203125, -0.81884765625, -0.7890625, -1.33203125, -0.27392578125, 0.02252197265625, -0.4072265625, 0.199951171875, -0.8720703125, -0.1590576171875, -0.2225341796875, -0.443359375, -0.2086181640625, -1.2626953125, -0.576171875, 0.6064453125, -0.370849609375, -0.359619140625, -1.056640625, -0.900390625, -0.020233154296875, -0.5556640625, 0.56689453125, 0.348388671875, 1.103515625, -1.2255859375, 1.15234375, -0.69580078125, -0.77197265625, 0.646484375, 0.044830322265625, 1.0595703125, 0.74267578125, 1.201171875, 0.384033203125, 0.369140625, -1.173828125, 0.30859375, 0.1756591796875, -1.91796875, -0.91162109375, 1.1279296875, 0.0252532958984375, 1.99609375, -0.8193359375, 0.6357421875, 1.7470703125, -1.1376953125, -0.9296875, 0.62841796875, -1.0517578125, 1.9033203125, -0.1868896484375, -1.431640625, 0.9072265625, -1.00390625, -0.1380615234375, 0.93896484375, 0.923828125, -1.0576171875, -1.236328125, 0.30810546875, -0.258544921875, -0.66259765625, 1.1416015625, -0.0948486328125, -1.7333984375, 0.291748046875, 1.3095703125, -0.67138671875, -1.0146484375, 0.5283203125, 0.64697265625, 0.451416015625, 1.744140625, -0.426513671875, 0.73388671875, 0.82861328125, 0.11907958984375, 0.0771484375, -2.7109375, 1.083984375, 1.13671875, -0.82958984375, 0.4853515625, -0.4326171875, 1.1796875, 0.5947265625, -1.4033203125, 0.09954833984375, 0.1807861328125, -0.001499176025390625, -0.517578125, 1.677734375, 0.30908203125, 0.76513671875, 0.63037109375, 0.381591796875, 0.388916015625, -1.5458984375, -0.78955078125, 0.0499267578125, -0.8134765625, 1.146484375, -0.50732421875, -0.148681640625, 0.370361328125, 0.344482421875, 0.71240234375, 0.185546875, 0.3232421875, -1.587890625, 0.5869140625, 0.88427734375, -0.6943359375, -0.64892578125, 0.1785888671875, -0.356689453125, -0.453857421875, -1.3818359375, 0.028106689453125, 0.76123046875, 0.56787109375, 0.0943603515625, 0.258056640625, 2.740234375, 0.055938720703125, 1.0849609375, 0.4580078125, -1.7490234375, -0.04412841796875, -0.52197265625, 1.6494140625, 0.91845703125, -2.263671875, -1.353515625, -0.9794921875, -2.0703125, 0.59912109375, -0.77294921875, -0.57861328125, -1.234375, -0.556640625, 0.0254974365234375, -0.378662109375, 0.0036792755126953125, -1.341796875, 1.150390625, -0.434814453125, -0.392822265625, 0.521484375, 0.8037109375, -0.18017578125, -0.474365234375, 1.6767578125, -0.73583984375, -1.693359375, 0.07696533203125, -0.1519775390625, -0.23486328125, 0.435546875, 1.7861328125, -0.7890625, -1.30078125, 2.1640625, 1.115234375, 0.445556640625, 1.140625, 0.50537109375, 2.279296875, 2.544921875, -0.212646484375, 1.4189453125, 1.7490234375, -1.490234375, -1.5966796875, -0.384033203125, -0.2435302734375, 0.7626953125, 1.73046875, -1.2373046875, 0.96826171875, 1.255859375, -1.1669921875, -1.412109375, 1.78125, -0.54541015625, 0.052764892578125, 0.451904296875, -0.2320556640625, -0.681640625, -0.3359375, -2.240234375, -0.1976318359375, 0.2308349609375, -0.568359375, -1.0615234375, -0.37158203125, 0.5205078125, -0.11572265625, 2.1640625, -0.4853515625, -0.1263427734375, -1.236328125, 1.697265625, 0.0513916015625, -0.257568359375, -0.8876953125, -1.0341796875, -0.248046875, -1.0888671875, 0.1971435546875, -0.66943359375, 0.06365966796875, -1.1787109375, -1.4091796875, -0.54638671875, 1.234375, 0.5439453125, 0.0218505859375, 1.29296875, 0.005016326904296875, -0.07861328125, -0.5478515625, 0.126708984375, -0.298828125, 0.15966796875, -0.6533203125, 1.978515625, 0.305908203125, 1.5888671875, 1.3134765625, -2.115234375, -1.826171875, -1.830078125, -0.48193359375, -1.294921875, -0.91650390625, 1.0361328125, 0.7294921875, -0.216796875, -0.2205810546875, 0.716796875, 1.7548828125, 0.6103515625, -0.375732421875, -0.0880126953125, -0.1365966796875, -0.87109375, 0.31689453125, 0.028961181640625, 0.36865234375, 0.22607421875, -1.0947265625, 0.2254638671875, -0.51611328125, -0.873046875, 0.342041015625, 1.427734375, -1.7265625, -0.72607421875, 0.8251953125, 0.54638671875, -0.54248046875, 0.86328125, -0.4384765625, 0.0821533203125, 0.63720703125, -1.326171875, 1.0537109375, 0.96630859375, -1.28515625, 0.1695556640625, 1.30078125, 0.40478515625, 0.513671875, -0.346435546875, 0.379638671875, 0.78271484375, -1.4658203125, -1.216796875, 0.454833984375, -0.292724609375, -2.57421875, -0.422119140625, 0.91064453125, 0.06683349609375, 0.168701171875, 0.50830078125, 0.1978759765625, -0.171142578125, 2.416015625, 0.340576171875, 0.56201171875, 0.431884765625, 0.486083984375, 1.4287109375, 1.6328125, 1.3876953125, 0.546875, 0.42626953125, -1.828125, -1.2548828125, 0.70751953125, 0.55908203125, 0.6767578125, -0.24609375, 2.44140625, -0.67138671875, -0.3603515625, -0.04315185546875, 1.66796875, -0.00164031982421875, 3.486328125, -0.78564453125, -0.408203125, -1.525390625, -0.09442138671875, 1.576171875, 1.193359375, 2.10546875, -1.97265625, 0.1292724609375, 0.226318359375, 0.0228729248046875, 0.2200927734375, 0.436767578125, 0.6953125, 0.611328125, 0.77099609375, 0.77197265625, 0.06561279296875, 0.116455078125, 0.43212890625, 0.70751953125, 0.07806396484375, -0.457275390625, 0.3095703125, 0.25927734375, -1.322265625, -1.4091796875, -0.0134429931640625, -0.419189453125, 1.1181640625, -0.159912109375, 0.01751708984375, 0.04241943359375, 0.37353515625, -1.53515625, -1.48828125, -1.02734375, 0.62060546875, -0.5791015625, -1.4365234375, -0.09283447265625, 1.1416015625, 0.46044921875, -0.80224609375, 0.6435546875, -0.46630859375, 0.25634765625, 0.859375, 1.181640625, -0.9013671875, -0.66015625, 0.97998046875, 1.41796875, 0.399169921875, 0.468505859375, 0.2061767578125, -0.87109375, 1.412109375, -0.225830078125, 0.97998046875, 0.188232421875, -0.82421875, 0.7587890625, -0.830078125, -1.44140625, 0.387939453125, 0.71142578125, 0.99658203125, 1.10546875, -0.458740234375, 0.017974853515625, 1.26171875, 1.634765625, 0.016510009765625, -1.0380859375, 0.1702880859375, -0.303955078125, -1.041015625, -0.0814208984375, 0.2705078125, -0.90576171875, -0.587890625, 1.6298828125, -0.0899658203125, 0.8173828125, -1.5693359375, 0.3486328125, -0.96728515625, 0.58740234375, -1.8916015625, -0.9560546875, 0.258544921875, -0.34423828125, 2.17578125, -0.79248046875, 1.326171875, -0.53271484375, -0.81787109375, -0.357177734375, -0.31982421875, 0.12481689453125, -0.43505859375, -0.79541015625, 0.56982421875, 1.3271484375, -0.8583984375, 1.146484375, 0.4111328125, -3.55078125, -0.052581787109375, -0.68603515625, -0.35791015625, -2.740234375, 0.76171875, -1.1669921875, 0.70166015625, 0.2421875, 1.546875, 0.421630859375, 2.9296875, -0.484619140625, -0.01267242431640625, 1.0341796875, -1.4326171875, 0.44091796875, 2.564453125, -0.28076171875, -1.4189453125, -0.6240234375, 0.330078125, 0.392578125, -1.091796875, 0.81689453125, -0.2406005859375, -0.51611328125, -1.072265625, 1.0615234375, -1.4833984375, -1.90234375, 0.05316162109375, 0.72802734375, -0.018585205078125, 0.82763671875, 0.10479736328125, -0.9169921875, -0.32080078125, -0.93359375, -1.0341796875, 2.044921875, 0.043487548828125, 1.23828125, -1.8232421875, -1.2978515625, -2.255859375, -0.8359375, 0.5595703125, 1.2783203125, 1.2783203125, -1.1259765625, 0.7470703125, -1.1884765625, 0.88232421875, 1.4189453125, -0.896484375, -0.8310546875, 0.138916015625, -1.810546875, 0.166259765625, -0.5078125, 0.119873046875, 1.3994140625, 0.75634765625, -1.607421875, 0.1722412109375, -0.11431884765625, -0.421630859375, -0.140869140625, 0.40283203125, -0.335205078125, 2.0703125, -0.1290283203125, 0.7060546875, -0.12078857421875, -1.28515625, 0.400634765625, 0.4912109375, 1.4697265625, -1.4970703125, 1.080078125, -0.251953125, 0.8203125, 0.79443359375, -0.100830078125, 0.80859375, 0.38427734375, -2.892578125, -0.58203125, -0.56787109375, -0.394287109375, 0.98974609375, -0.157470703125, -0.97412109375, 1.99609375, -0.194091796875, -0.1610107421875, -0.89404296875, -0.281005859375, -1.234375, 0.68798828125, -0.027587890625, 0.4091796875, -0.238525390625, 0.180908203125, 1.3828125, -0.56201171875, -0.8046875, 1.23046875, -1.2333984375, -0.030731201171875, -0.609375, 0.73876953125, 1.8974609375, 1.4736328125, 1.3232421875, 0.060516357421875, -0.59130859375, 0.3525390625, -1.1181640625, -0.3115234375, -0.004764556884765625, 1.666015625, -1.326171875, -0.387939453125, 0.1922607421875, -1.1201171875, 0.80322265625, 0.8193359375, -0.5234375, -0.98291015625, 0.299560546875, 0.53466796875, -0.484375, -1.5830078125, -0.322998046875, -1.3935546875, -1.4765625, 0.15478515625, 0.283203125, -0.10906982421875, -1.01953125, 0.84423828125, 1.505859375, 0.58544921875, 0.83837890625, -0.7001953125, -0.303466796875, -0.83544921875, -0.3271484375, -0.1722412109375, -0.81103515625, 0.27880859375, 1.0517578125, -0.2095947265625, 1.30078125, 1.3681640625, 1.0556640625, 0.183837890625, -0.8896484375, -0.826171875, 0.01204681396484375, -0.90576171875, 0.73388671875, 1.01953125, 1.419921875, -0.70751953125, 0.1988525390625, -0.258056640625, 0.88671875, 0.408935546875, -1.099609375, -1.244140625, -0.86572265625, 0.248779296875, -0.1396484375, -2.751953125, 1.7431640625, -0.0828857421875, 0.296142578125, -0.462158203125, 1.724609375, 1.6982421875, 0.252685546875, -0.89111328125, -0.042236328125, 0.4365234375, 0.46826171875, 0.65966796875, -0.8173828125, -1.818359375, -1.5791015625, -0.343994140625, 3.162109375, -1.0263671875, 1.19921875, 0.73681640625, 0.70068359375, 1.8427734375, -0.25634765625, 0.1734619140625, 0.439453125, 0.572265625, -0.13232421875, 0.402099609375, 0.5810546875, -1.791015625, 0.059844970703125, -0.266357421875, -2.22265625, 0.98388671875, -2.318359375, -0.60009765625, 2.423828125, 1.3203125, -0.88330078125, -0.1875, -1.0234375, -0.4736328125, 1.947265625, -1.1318359375, -1.3271484375, 1.0322265625, -1.2490234375, 1.2978515625, 0.159423828125, -1.0234375, -0.66455078125, -2.25, -0.74951171875, 1.6396484375, 0.278076171875, -0.72900390625, -0.78076171875, 2.2734375, -0.6015625, 0.1600341796875, 0.0999755859375, 0.000667572021484375, -0.6005859375, 0.79248046875, -0.328369140625, -0.37890625, -0.18017578125, -1.7666015625, -0.6015625, -0.5966796875, 0.28857421875, 0.5791015625, 0.1702880859375, -0.4052734375, 0.1961669921875, -0.6962890625, 0.87255859375, 0.87890625, 0.6591796875, -0.52978515625, 0.1619873046875, -0.089599609375, 0.07177734375, 0.70654296875, 0.128173828125, -0.016448974609375, 0.99853515625, -0.058868408203125, 0.87451171875, -2.296875, -0.01116180419921875, 0.124267578125, 1.478515625, 0.79833984375, -1.5966796875, -0.97705078125, -0.285400390625, 2.939453125, 0.415283203125, 1.0966796875, -0.955078125, 0.607421875, 1.6630859375, 0.8486328125, 2.18359375, -0.79833984375, -2.234375, 1.0517578125, 2.20703125, 0.11761474609375, -1.3623046875, -0.8876953125, 0.480224609375, 0.62158203125, -0.4658203125, 0.884765625, 0.46630859375, 1.9736328125, 0.71240234375, -1.0947265625, 0.1007080078125, 0.49267578125, -0.9140625, -0.0360107421875, -1.1474609375, -0.85400390625, -0.82861328125, 0.72021484375, 0.61962890625, 0.111328125, 1.2900390625, 0.07427978515625, 0.1072998046875, -0.223876953125, -1.693359375, 0.70947265625, 1.03125, -1.1474609375, 1.130859375, -1.1845703125, 0.485107421875, 0.6259765625, -1.224609375, 0.284912109375, 0.6904296875, -0.61572265625, -1.3251953125, 0.5869140625, 0.6904296875, -0.85791015625, -0.1949462890625, 0.52294921875, -0.81640625, 0.70751953125, 1.67578125, -1.1884765625, 0.5791015625, 0.390380859375, -0.84765625, -0.8583984375, 0.5908203125, -0.6142578125, 1.5341796875, 0.133056640625, -0.677734375, -0.1656494140625, -0.1090087890625, -1.2587890625, -1.0673828125, -1.3388671875, 0.94580078125, 1.0107421875, 1.0615234375, -1.564453125, 0.673828125, 3.1953125, 0.78173828125, -0.4013671875, -0.68896484375, 0.1253662109375, -0.65087890625, -0.1903076171875, -0.1304931640625, -0.54541015625, 0.57763671875, -0.5556640625, -0.1153564453125, 0.8642578125, 0.46435546875, -1.87109375, -0.47705078125, 0.2415771484375, 0.06048583984375, 1.4755859375, -0.353759765625, 0.2257080078125, 0.9814453125, -1.712890625, 1.1650390625, 0.8603515625, 0.84814453125, -1.2275390625, 0.96142578125, 0.623046875, 0.57470703125, 0.7158203125, -0.990234375, -0.468505859375, 1.7177734375, -0.257080078125, 0.44287109375, -0.65478515625, 0.438720703125, 1.7451171875, -0.150146484375, 1.1416015625, 0.5146484375, 1.0107421875, 0.2109375, 0.4580078125, -0.71728515625, 1.3583984375, -1.443359375, -0.703125, -1.2431640625, -3.75, 0.1676025390625, 0.03131103515625, -0.892578125, 0.9189453125, -0.246337890625, -1.126953125, 0.251953125, -1.5, -0.6708984375, 0.311279296875, -0.7568359375, -1.173828125, -2.173828125, -0.427490234375, 0.283447265625, 1.1064453125, 0.9482421875, 2.0078125, 1.01953125, 0.20751953125, -0.96875, 2.4296875, 0.40673828125, -1.2548828125, -0.40234375, -0.9892578125, -1.2880859375, 0.0477294921875, 0.053192138671875, -0.5625, -1.5625, 0.271240234375, 1.6025390625, 0.93896484375, -0.499755859375, -0.194091796875, 1.46875, 2.66796875, 1.306640625, 0.4921875, 0.30029296875, -1.068359375, -0.3154296875, -0.640625, 1.5009765625, -1.2724609375, 0.806640625, 2.25390625, 0.223876953125, 0.57666015625, -0.042236328125, 0.78271484375, 0.6357421875, 0.919921875, 0.65869140625, 0.91357421875, -0.214111328125, 0.3837890625, 0.9599609375, -0.417236328125, -0.6884765625, 0.177490234375, 0.0848388671875, -0.28466796875, -0.3671875, -0.2364501953125, -0.66650390625, -0.83544921875, -0.66015625, -1.5224609375, -0.4990234375, -0.2213134765625, -0.80712890625, -0.07818603515625, 0.8857421875, 0.75048828125, -0.87744140625, -0.06304931640625, 0.8740234375, 0.7451171875, 0.4765625, -0.290771484375, 0.36181640625, -0.912109375, 0.40185546875, -0.410888671875, 0.348876953125, -0.039154052734375, 0.310302734375, 1.2158203125, 0.95166015625, 0.434814453125, -1.54296875, 0.0206451416015625, -1.248046875, 1.05859375, 0.1888427734375, 0.779296875, 1.2099609375, -0.06903076171875, -0.6845703125, -0.194580078125, -0.07342529296875, -0.7392578125, 1.595703125, -1.2734375, -1.1689453125, 0.6162109375, -0.79736328125, -0.603515625, -0.392333984375, -2.13671875, 2.314453125, 1.029296875, -0.325927734375, 0.062286376953125, -0.304443359375, 0.64990234375, -0.90234375, 1.583984375, 0.79833984375, 0.92724609375, -0.76806640625, 0.95947265625, -1.0419921875, -0.03448486328125, 0.04296875, -1.1982421875, -1.294921875, -0.39990234375, -1.7158203125, -2.25390625, -0.06390380859375, -1.876953125, 1.3173828125, -0.12939453125, -0.73681640625, -0.6142578125, -0.349853515625, -1.0126953125, 0.185546875, -0.56201171875, 0.56591796875, 1.265625, 0.212158203125, -0.07305908203125, -1.6083984375, 1.267578125, -0.18017578125, 1.216796875, 0.560546875, 0.2237548828125, -0.400146484375, -0.312744140625, -0.86474609375, -1.2685546875, -1.4755859375, 2.3125, 1.1533203125, 0.8828125, -0.491943359375, -1.234375, 1.1162109375, -0.439208984375, 2.5234375, 0.8037109375, 1.0478515625, -1.20703125, 1.4345703125, 0.10498046875, -1.4111328125, -0.449462890625, 0.046051025390625, 1.583984375, 0.07958984375, -1.509765625, -0.79736328125, 1.1904296875, 0.432861328125, 0.81689453125, -0.0165557861328125, -1.5634765625, -1.2021484375, -0.34423828125, 2.107421875, 0.052764892578125, 1.2060546875, 0.292236328125, 1.8837890625, 0.002643585205078125, -0.67333984375, 0.379150390625, 1.0732421875, 0.04608154296875, 0.826171875, -0.48486328125, 0.0096588134765625, -0.603515625, -1.5107421875, 1.2421875, -1.369140625, 1.2119140625, 2.5625, 0.96728515625, 1.169921875, 0.93359375, -0.4560546875, -0.30029296875, 2.02734375, -0.9638671875, 0.303955078125, -0.57763671875, -0.10968017578125, 0.30810546875, 0.90283203125, -0.751953125, 0.978515625, 1.0859375, -0.63623046875, 0.7490234375, 0.2440185546875, 0.436767578125, 0.389892578125, 0.6728515625, -0.53662109375, -0.1435546875, 1.31640625, 0.67333984375, -1.1240234375, 0.50390625, -0.521484375, 0.318115234375, -0.1923828125, 0.5595703125, 1.166015625, 1.2158203125, 0.810546875, -0.21337890625, -0.69482421875, 0.1666259765625, -0.7861328125, 0.5361328125, 0.4609375, 0.66357421875, -1.638671875, 0.6279296875, -0.289306640625, -0.08258056640625, 0.04718017578125, 0.85546875, -1.1416015625, -1.1474609375, 0.47802734375, -0.30224609375, -1.3466796875, 1.3076171875, 0.430419921875, -0.494384765625, 1.92578125, -2.130859375, -0.020660400390625, -0.919921875, 0.8984375, -1.326171875, -0.966796875, 0.83154296875, 0.03814697265625, 0.6474609375, 0.51806640625, -1.7783203125, 2.5, 0.1524658203125, 0.578125, 0.96826171875, 0.873046875, -0.281494140625, -0.164794921875, -1.0068359375, -0.81884765625, 0.80615234375, 0.274658203125, 0.453125, -0.06060791015625, 0.051483154296875, -0.35009765625, -1.900390625, -0.9521484375, -0.2198486328125, -0.1806640625, -0.370849609375, -1.099609375, 0.396728515625, -1.98046875, -0.02374267578125, 0.412353515625, -0.40283203125, 0.9111328125, 0.62841796875, -0.63232421875, -1.8291015625, 1.6982421875, 0.5576171875, 1.3974609375, 0.389404296875, 0.2210693359375, -0.72021484375, 0.8759765625, 0.10687255859375, 0.157470703125, -0.423583984375, 0.09613037109375, -1.1083984375, 0.54736328125, 0.2381591796875, -1.73828125, -1.916015625, -1.0390625, 0.72705078125, -1.171875, 0.6279296875, -1.63671875, -0.58740234375, -0.96826171875, 0.178466796875, -0.1441650390625, 1.8916015625, 0.04327392578125, 0.437255859375, 2.416015625, 1.3408203125, 2.30078125, 0.0236663818359375, 0.51513671875, 0.1669921875, 0.03338623046875, 0.5927734375, 0.68505859375, -0.8857421875, -0.22509765625, 0.29052734375, 2.162109375, 0.65380859375, 0.159912109375, -0.395263671875, -1.36328125, 2.17578125, 1.2109375, -1.0712890625, 2.19140625, 0.5205078125, -0.0758056640625, -0.117919921875, -0.217529296875, -0.58740234375, -0.865234375, -0.89013671875, -0.055023193359375, 1.65625, 0.76318359375, 0.513671875, -1.900390625, -1.4267578125, 1.3984375, 0.277099609375, -1.310546875, 0.65283203125, -0.69091796875, 1.2958984375, 0.8876953125, -0.64453125, -0.055450439453125, 0.6162109375, -1.4970703125, 0.8828125, -0.78759765625, -0.734375, 0.552734375, -0.92236328125, 1.1181640625, -0.11688232421875, 0.49072265625, 0.96240234375, 0.470947265625, -0.66650390625, -0.89794921875, -1.830078125, 1.2626953125, -1.828125, -0.3984375, -0.59130859375, -0.70751953125, 0.9091796875, -1.8701171875, -0.40234375, -1.28515625, -1.0927734375, -0.208740234375, -1.0078125, 0.055389404296875, -0.2060546875, 0.83935546875, -0.65673828125, -0.810546875, -0.6142578125, -0.79248046875, -0.6123046875, -1.109375, 1.3974609375, -0.61572265625, 0.59912109375, -0.017852783203125, 0.24658203125, -1.0029296875, -0.472900390625, 0.78076171875, 0.39111328125, -1.6298828125, 1.48828125, 0.40380859375, 0.0109405517578125, 1.349609375, 0.37890625, 0.09527587890625, -0.48974609375, -0.295166015625, -0.392333984375, -0.43017578125, 1.408203125, 1.580078125, 0.20703125, -0.0477294921875, 0.056121826171875, -1.0869140625, -1.1376953125, 1.486328125, -0.58203125, -0.5654296875, -1.46875, 0.66650390625, -0.171142578125, -0.59326171875, -1.5244140625, 0.326416015625, 0.62109375, -1.5302734375, -0.213134765625, -0.294921875, -1.98046875, 0.10296630859375, -1.2216796875, 0.8505859375, 0.61279296875, -0.08416748046875, 1.953125, -1.0546875, -1.2265625, 1.232421875, 0.9853515625, 0.87548828125, 0.012847900390625, -0.8671875, 1.59375, 1.2421875, 1.4658203125, 1.037109375, -0.1689453125, 1.478515625, -0.44580078125, 1.322265625, -0.81787109375, 0.73779296875, 0.2479248046875, 0.049407958984375, -0.103515625, 0.64013671875, 0.09857177734375, 1.904296875, -0.1373291015625, 0.126220703125, -0.82080078125, -1.666015625, -0.278564453125, -1.9228515625, -0.3427734375, 0.19921875, 0.30126953125, 0.7109375, -1.3994140625, 1.2158203125, 0.1854248046875, 0.339111328125, -0.308349609375, 1.6455078125, -0.94091796875, -0.265625, 0.26513671875, 0.216064453125, -1.201171875, -0.049285888671875, -1.0869140625, 0.44970703125, 0.496826171875, 1.9384765625, 1.0263671875, -0.95849609375, -0.55078125, -2.283203125, 0.307861328125, 0.12060546875, -0.068359375, -0.60693359375, -0.125732421875, -0.5810546875, 0.46435546875, 1.4140625, -0.931640625, 0.377197265625, 0.77734375, 0.607421875, -0.6005859375, 1.41015625, 1.83203125, -0.396240234375, 1.677734375, 0.80859375, 1.6796875, -0.293212890625, 1.6982421875, 0.525390625, -1.419921875, -0.6171875, -0.443603515625, -1.13671875, -1.0048828125, -0.426025390625, 0.09521484375, 0.445556640625, -0.27392578125, -1.298828125, -0.1927490234375, -0.0136566162109375, 0.80322265625, 0.014373779296875, -1.20703125, -0.5537109375, 1.5625, -1.12890625, 0.19189453125, -0.86767578125, -0.728515625, 0.0894775390625, -0.51318359375, 1.8349609375, 1.8525390625, -0.8486328125, 0.006633758544921875, 0.80224609375, 1.603515625, -0.6279296875, 0.2283935546875, 0.78564453125, -0.499755859375, 2.32421875, 0.8037109375, 1.7880859375, 0.97412109375, -0.6650390625, 0.0887451171875, -1.0537109375, -0.833984375, -0.884765625, 0.394287109375, 1.09375, -1.513671875, 0.047607421875, -0.4560546875, -0.361083984375, 0.73193359375, 0.6484375, -0.89453125, 0.9501953125, -0.401123046875, 0.287841796875, -0.50537109375, -0.7724609375, 1.056640625, 0.08636474609375, -1.12890625, 0.381591796875, -0.6142578125, -0.05914306640625, 0.406982421875, 1.61328125, -0.619140625, -0.91552734375, 0.779296875, -0.275390625, 1.9248046875, 0.34521484375, 0.491455078125, -1.5791015625, -0.501953125, -0.0163726806640625, 1.3564453125, 0.603515625, 0.1451416015625, -0.9111328125, 0.525390625, -0.3291015625, -0.288818359375, 2.109375, -1.060546875, 0.86376953125, -0.58837890625, 0.002895355224609375, 0.84814453125, -0.320556640625, 1.1005859375, 1.041015625, -0.413818359375, -1.244140625, 1.0654296875, 0.05621337890625, -1.5419921875, 0.87841796875, 0.80126953125, 0.11651611328125, -1.28125, 1.201171875, 0.34228515625, 1.755859375, -0.0130462646484375, 0.33544921875, 0.1478271484375, 0.7431640625, -0.62548828125, -1.4482421875, -0.259521484375, -0.1307373046875, -0.515625, 0.845703125, -0.411865234375, 0.86962890625, 1.7275390625, 0.20458984375, 0.50927734375, -1.59375, -2.615234375, -0.10736083984375, 1.0205078125, 0.27392578125, 0.75732421875, -1.28125, -1.3037109375, -0.54638671875, 0.576171875, -1.4599609375, -0.07427978515625, 1.2412109375, -0.11920166015625, -0.52490234375, 0.869140625, 0.2919921875, -0.210205078125, 0.460205078125, -0.26611328125, 0.52001953125, 0.1630859375, 0.1932373046875, -0.447265625, 0.142822265625, -0.6435546875, -0.888671875, 0.04486083984375, -1.4482421875, 0.2135009765625, -0.364990234375, 0.2005615234375, 0.225341796875, 0.95166015625, -0.876953125, 0.0765380859375, -0.06353759765625, -1.6630859375, -0.30126953125, -1.4013671875, -0.0245361328125, -0.469482421875, -0.26171875, 2.3515625, -0.94580078125, 1.4013671875, -0.052398681640625, 0.2183837890625, 0.413818359375, 0.08453369140625, -1.2080078125, 0.603515625, -0.5419921875, -1.560546875, -0.41650390625, 0.57861328125, 0.2210693359375, -0.492919921875, 0.90380859375, 0.94287109375, -1.60546875, 1.5771484375, -0.80517578125, -1.2314453125, -0.873046875, 0.460693359375, 0.493896484375, -1.7353515625, -0.82421875, -1.1083984375, -0.97216796875, -0.0662841796875, -2.529296875, -0.477294921875, -0.54736328125, -0.9384765625, 0.29150390625, 0.671875, 0.55810546875, 0.701171875, 0.341064453125, -0.0222625732421875, -0.0081329345703125, 0.1322021484375, -0.279296875, -0.35986328125, 1.828125, 0.6953125, 0.1949462890625, -1.6474609375, -1.2587890625, 3.02734375, 0.453125, 0.127685546875, 1.25390625, -1.4375, 1.486328125, -0.572265625, -0.45703125, -0.155517578125, 1.517578125, 1.669921875, 1.76953125, -0.0615234375, -0.455322265625, -0.2083740234375, -0.931640625, -0.03497314453125, 0.235107421875, 0.460693359375, 0.73388671875, 0.63623046875, 0.07012939453125, 0.91552734375, 1.9931640625, 1.318359375, -1.142578125, 0.00396728515625, -0.84912109375, -0.6171875, 1.6943359375, 0.703125, -1.35546875, 0.513671875, -1.029296875, -0.88427734375, 0.340576171875, 1.884765625, -0.587890625, 1.0634765625, 0.5390625, 2.794921875, 0.9599609375, 1.5927734375, -0.4130859375, 0.489990234375, 2.72265625, 0.4296875, -1.85546875, -1.958984375, -1.0068359375, -0.0927734375, -0.1654052734375, -1.8623046875, -1.568359375, -1.5390625, 0.141845703125, 1.0888671875, -2.373046875, -0.195556640625, 0.0667724609375, 0.32470703125, 0.64111328125, -0.30224609375, -0.64404296875, -0.50927734375, -0.034423828125, -0.376953125, -1.115234375, 1.5166015625, -1.51171875, -0.71484375, 0.09619140625, -2.123046875, -0.07427978515625, 0.19091796875, 0.390380859375, -1.3603515625, -0.421875, 0.68505859375, -0.56298828125, -0.061492919921875, -1.474609375, -0.783203125, 0.88232421875, 0.521484375, -1.55078125, 1.7900390625, -0.45751953125, -0.5078125, -0.01873779296875, -1.251953125, -0.264892578125, -0.744140625, 0.9921875, -1.765625, 1.5322265625, -0.943359375, 2.255859375, 1.09765625, -0.79248046875, 0.1279296875, 1.0537109375, -0.73486328125, 2.833984375, -0.62548828125, -1.5595703125, 1.1279296875, 0.446044921875, -0.1541748046875, 1.8583984375, -1.619140625, 1.1513671875, 2.00390625, -0.44921875, 0.11895751953125, 0.2193603515625, -1.2529296875, -0.892578125, 2.53515625, -0.71826171875, 1.298828125, 2.619140625, 0.447265625, 1.201171875, 1.42578125, 1.2265625, 0.544921875, -0.331298828125, -1.1787109375, -2.0625, -0.314453125, -1.30078125, 0.480224609375, -2.162109375, -0.405517578125, -1.1904296875, 1.16015625, -0.03472900390625, 0.509765625, 0.935546875, -0.138427734375, 0.324951171875, 0.050506591796875, 1.392578125, -1.384765625, 1.224609375, 0.06695556640625, 0.2021484375, -1.935546875, 1.2099609375, 1.373046875, -0.34619140625, 0.1773681640625, 0.65283203125, 0.82177734375, -0.49072265625, -0.09423828125, 1.8671875, 0.1312255859375, 1.5927734375, 0.467041015625, -1.5166015625, -0.70068359375, -0.55517578125, 0.430908203125, 0.94384765625, -1.7509765625, 0.9013671875, 1.1396484375, -0.225830078125, 0.94287109375, -0.32666015625, -0.85546875, -0.59619140625, 0.1356201171875, 1.154296875, -0.5322265625, -0.28564453125, 1.201171875, -0.27490234375, -2.478515625, 0.482666015625, 0.42431640625, -1.5126953125, -0.50732421875, -0.1793212890625, -1.3369140625, -0.86376953125, 0.8955078125, 0.0219879150390625, -0.5078125, 0.263427734375, 0.6748046875, 1.228515625, -2.107421875, -0.48095703125, -0.66162109375, 0.05712890625, -1.1435546875, 1.1728515625, 1.173828125, 0.6484375, -0.1092529296875, -1.91015625, 0.118896484375, 0.391357421875, 0.62060546875, 0.6943359375, 0.1654052734375, 0.962890625, -0.110595703125, -1.5908203125, 0.86474609375, 0.264892578125, -0.344970703125, 0.537109375, -0.2384033203125, -0.08013916015625, -1.171875, -0.166259765625, 1.0693359375, 0.33642578125, -0.546875, -0.609375, 0.82373046875, -0.6240234375, -1.28125, 0.244384765625, 0.8173828125, -0.5673828125, -0.80615234375, -0.440673828125, -0.260009765625, 0.077880859375, -1.72265625, -0.9140625, -0.72705078125, -0.80029296875, 0.7236328125, 0.93115234375, 0.92236328125, 1.3134765625, -1.7568359375, 0.466064453125, 0.0791015625, 0.44482421875, -0.78564453125, 0.0254364013671875, 0.385009765625, 0.42138671875, 0.1636962890625, 0.4228515625, -0.16650390625, -1.2939453125, -1.36328125, 1.6611328125, -0.5830078125, 0.54248046875, -1.39453125, 0.030364990234375, -1.244140625, 0.4580078125, -1.646484375, -0.814453125, -1.4970703125, -0.8359375, -0.360595703125, 1.8330078125, -0.310546875, 0.01398468017578125, 0.51806640625, -0.1611328125, 0.02996826171875, -0.857421875, 0.3935546875, -0.68896484375, -0.84130859375, -0.31201171875, -0.02288818359375, 1.3037109375, 0.6162109375, 0.7119140625, -0.78857421875, 0.325927734375, -0.2744140625, -0.97705078125, 0.64599609375, -0.1240234375, -0.165771484375, 1.02734375, 0.1141357421875, 0.22900390625, 0.8720703125, 0.0330810546875, 0.281982421875, 0.0013113021850585938, 0.56689453125, 1.6103515625, -1.1328125, 0.039337158203125, 1.666015625, -0.6630859375, -0.2666015625, -0.52001953125, -1.1494140625, -0.156982421875, -0.7919921875, -0.193115234375, 1.208984375, -1.8349609375, 0.499267578125, 0.93115234375, -0.8857421875, -0.099853515625, 0.8291015625, 1.3896484375, -0.7568359375, -0.98779296875, -0.30126953125, -1.5615234375, -0.03680419921875, 1.8369140625, -1.619140625, -2.232421875, -0.875, 0.658203125, 0.5556640625, -0.45458984375, 0.15625, 0.96826171875, 0.30029296875, 1.8310546875, -0.57861328125, 0.57177734375, -0.935546875, -0.441650390625, -0.6708984375, -0.366943359375, 1.2578125, -1.060546875, -0.6787109375, -0.51123046875, 0.165283203125, -0.63818359375, -0.45068359375, -1.009765625, 0.466064453125, 0.88037109375, 2.14453125, -0.50341796875, 0.265869140625, -0.96240234375, 1.0146484375, 2.169921875, 0.376220703125, 0.426513671875, -0.0283966064453125, -1.60546875, 0.724609375, 0.7705078125, 0.5556640625, 1.158203125, -0.01534271240234375, -0.0533447265625, 0.56201171875, -1.607421875, -0.0034427642822265625, 1.345703125, 1.0732421875, 0.94775390625, -0.359130859375, 1.013671875, 0.5869140625, -0.6982421875, 0.125244140625, 1.353515625, -1.1767578125, 0.37646484375, 0.6650390625, -0.712890625, 1.3955078125, -0.8076171875, 0.215087890625, 1.6552734375, 0.56298828125, 1.326171875, -0.487060546875, -1.0439453125, 1.1259765625, -0.38818359375, 2.646484375, -0.35986328125, -1.51953125, -0.2457275390625, -1.4755859375, 1.1025390625, 0.20263671875, -0.86572265625, 0.6611328125, 0.09844970703125, -0.137939453125, 2.150390625, 1.1728515625, 0.423583984375, -0.6708984375, 0.07958984375, 1.3828125, -0.3701171875, -1.6591796875, -0.4033203125, 2.0703125, 1.2138671875, -0.62451171875, -1.9052734375, 0.791015625, -1.5087890625, 1.0419921875, 0.08203125, 0.701171875, 1.3056640625, 1.1181640625, -0.208740234375, -0.4580078125, 0.54736328125, 0.25927734375, -1.9970703125, 0.7451171875, 1.69921875, 0.330078125, 1.1982421875, 0.365966796875, 0.278564453125, -0.48828125, -1.55078125, -2.69921875, 1.0556640625, 0.46484375, 0.63330078125, 0.1971435546875, 1.41796875, 1.32421875, 0.04998779296875, -1.9404296875, 0.93017578125, -0.82958984375, -1.08203125, -1.2568359375, -1.80859375, 2.234375, -0.96337890625, -1.6328125, 0.15771484375, -0.2205810546875, -0.791015625, 0.916015625, 0.99462890625, 0.72021484375, 0.93603515625, -0.96142578125, -2.11328125, -1.3212890625, 0.174072265625, 0.94189453125, 1.6796875, -1.0263671875, 0.7138671875, -1.4091796875, -0.83935546875, 1.0498046875, 0.44140625, -0.23828125, 0.7490234375, 0.1790771484375, 1.8662109375, 0.6845703125, 0.1824951171875, 1.8818359375, -0.09014892578125, 0.52783203125, -0.484375, -0.982421875, 0.9287109375, 1.384765625, -0.54443359375, -0.517578125, 1.2216796875, -1.0126953125, 0.9384765625, 1.458984375, 0.564453125, -0.30859375, 1.994140625, -0.00429534912109375, 1.41015625, -1.96484375, 0.09344482421875, -1.1083984375, -0.487060546875, 0.42333984375, -0.4033203125, 0.75390625, -1.283203125, -1.712890625, -0.2313232421875, 1.205078125, -0.98779296875, 0.6396484375, 0.1959228515625, 0.84375, 0.70458984375, 0.53759765625, -1.6533203125, -0.02520751953125, -0.0777587890625, -0.299560546875, 0.662109375, 0.9287109375, 0.11578369140625, -0.91455078125, 0.1470947265625, 0.291015625, -0.1558837890625, -0.50732421875, -0.273681640625, 0.317138671875, 0.7783203125, -0.27490234375, -1.37109375, -1.9140625, 0.4892578125, -1.3115234375, -0.253662109375, 0.61767578125, 0.17822265625, -0.51953125, 1.0302734375, -0.548828125, 0.138671875, 0.47216796875, -0.7314453125, 2.34765625, -0.52587890625, 0.2021484375, -0.8955078125, -1.4521484375, -1.6572265625, -0.32861328125, 0.5498046875, -0.0789794921875, -0.25439453125, -0.810546875, 0.98486328125, 0.1376953125, 2.509765625, 1.0576171875, -2.478515625, -0.9931640625, 0.77099609375, -0.552734375, 1.26171875, -0.0823974609375, 0.68359375, 0.483642578125, 1.1298828125, 1.4033203125, -0.802734375, 2.484375, -1.2880859375, 0.64501953125, -0.0902099609375, -1.58203125, -0.80029296875, -0.282958984375, 0.720703125, 0.03375244140625, -1.1953125, -0.76806640625, -1.6669921875, 0.65234375, 0.77978515625, -0.82177734375, -0.41650390625, -0.0159759521484375, -0.84619140625, -1.748046875, -0.00714111328125, -0.97314453125, 1.5673828125, -0.03985595703125, 0.0601806640625, 0.58837890625, 0.26708984375, 2.26171875, 0.65625, 0.053924560546875, 1.287109375, 0.8134765625, 1.3056640625, 0.412353515625, -1.408203125, -0.334228515625, -1.5302734375, 1.0634765625, -1.439453125, 0.012969970703125, 0.0037078857421875, -1.708984375, -2.263671875, -0.24853515625, 1.310546875, 0.20458984375, 0.712890625, -1.443359375, 0.81298828125, -2.0859375, -1.4736328125, 0.5205078125, -1.8896484375, 0.66796875, -0.392333984375, -0.144287109375, -0.5537109375, -0.75146484375, -0.394775390625, -1.095703125, -1.443359375, 0.10675048828125, -0.48583984375, 0.7080078125, 0.35302734375, -2.140625, -0.65478515625, 1.2216796875, -1.46875, 0.264404296875, -0.037445068359375, -1.1875, -0.666015625, -1.0224609375, -1.203125, -0.27685546875, 0.4296875, -1.87109375, -0.1856689453125, 0.8759765625, 0.075439453125, -0.1317138671875, -0.0209503173828125, 0.366455078125, -0.57470703125, -0.51708984375, 1.2275390625, 1.46875, 0.0251617431640625, -1.048828125, -1.25390625, -0.798828125, 0.46533203125, 0.83544921875, 0.51806640625, -0.8671875, -0.446044921875, 0.83935546875, -1.71484375, 1.69140625, -0.451416015625, 0.0877685546875, -0.8232421875, 1.5654296875, -0.09326171875, 1.7470703125, -0.375, 0.1002197265625, 0.280517578125, -0.140869140625, 0.06036376953125, -0.04998779296875, 0.464111328125, 0.1419677734375, 1.8095703125, -0.8271484375, -1.193359375, 1.6015625, -1.2763671875, -1.212890625, 0.79736328125, 2.43359375, 0.2509765625, 0.5791015625, 1.4970703125, -0.3974609375, -0.0697021484375, -0.11541748046875, 0.56591796875, 0.513671875, 0.7490234375, 1.4912109375, 0.61279296875, 0.248779296875, -0.27880859375, 0.493408203125, 0.490478515625, 1.0048828125, 0.55615234375, 0.7607421875, 0.6005859375, -1.0732421875, 0.400390625, -0.63916015625, 0.01522064208984375, -0.1236572265625, -0.6689453125, -0.14306640625, 0.1527099609375, -0.168701171875, -0.1309814453125, 0.78466796875, -0.68798828125, 0.818359375, -0.368408203125, 0.32275390625, 0.13720703125, -0.61328125, -0.3125, -0.0599365234375, -1.7744140625, 1.64453125, -0.40380859375, -1.2763671875, -0.1751708984375, -0.92919921875, -1.546875, 0.1578369140625, 0.26171875, -0.98681640625, -0.13623046875, 0.0122833251953125, -0.25244140625, 0.220947265625, -0.7138671875, -1.59375, -0.0278778076171875, -0.63623046875, -1.1318359375, 1.0205078125, -1.06640625, 0.401123046875, -1.083984375, 0.1829833984375, 3.021484375, -0.22412109375, -0.174072265625, 0.923828125, -0.4599609375, 3.466796875, 0.425048828125, 0.293212890625, -0.7509765625, -0.580078125, 1.63671875, -1.2353515625, 1.224609375, 1.509765625, 1.2607421875, -1.4423828125, -0.7412109375, -0.1834716796875, 1.078125, 0.402587890625, 1.0966796875, 0.1431884765625, 0.6259765625, -1.23828125, 5.555152893066406e-05, -0.81787109375, -1.931640625, -0.0114593505859375, -0.271240234375, 1.052734375, -0.505859375, 0.85888671875, 1.2919921875, 2.18359375, 1.8984375, -0.06097412109375, 0.15380859375, -0.3955078125, -1.1259765625, -0.46484375, 0.047027587890625, 1.3486328125, -0.0305328369140625, 0.8779296875, 0.290771484375, -0.184814453125, 0.6884765625, 0.46826171875, 0.55322265625, -1.2607421875, -1.837890625, -0.8798828125, -0.160400390625, -0.73583984375, -0.73974609375, 0.9453125, -2.296875, 0.6123046875, -0.89208984375, -0.380126953125, -0.467529296875, 0.28173828125, 0.8056640625, -0.412353515625, -0.66162109375, 0.08465576171875, 1.3583984375, 0.62548828125, 0.5166015625, -1.064453125, -2.83984375, -0.1864013671875, 1.537109375, 2.6875, 0.236572265625, 0.07049560546875, -2.251953125, -2.02734375, -1.126953125, -0.64208984375, 1.4453125, -1.478515625, -1.052734375, -0.9501953125, -0.669921875, 0.7744140625, -0.64990234375, 1.3876953125, 0.78369140625, -0.55615234375, -0.82080078125, -1.3671875, -0.96142578125, -0.0243072509765625, -1.525390625, 1.6318359375, 0.93017578125, 0.736328125, 0.318359375, -2.8828125, -1.771484375, -0.6015625, 0.308837890625, 1.5888671875, -0.0100555419921875, -0.361572265625, 2.4375, 0.5400390625, -0.86181640625, -0.78173828125, 1.9208984375, 0.0911865234375, 0.5771484375, 0.35595703125, 2.005859375, 0.20458984375, -0.343994140625, -1.384765625, -1.298828125, -0.7822265625, 0.078369140625, -1.66796875, 1.3095703125, 0.440673828125, 0.13427734375, -1.0830078125, 0.64892578125, -1.0595703125, 0.2376708984375, 0.41845703125, 1.955078125, 2.322265625, 0.48095703125, -0.79833984375, 0.303466796875, -1.1015625, 1.294921875, 1.6640625, 1.46875, -2.09765625, 1.306640625, 1.3271484375, 0.92041015625, 1.052734375, -0.69189453125, 0.417724609375, 0.31103515625, -0.25439453125, -1.0068359375, -1.3095703125, 0.205810546875, -0.8603515625, -0.13671875, -2.841796875, -1.4580078125, -0.2313232421875, -0.051361083984375, 0.6455078125, 0.396240234375, 0.505859375, -0.59423828125, -0.485595703125, -1.111328125, -0.12408447265625, 0.9228515625, -0.1943359375, -1.4599609375, 1.197265625, -0.6455078125, 0.293701171875, 0.4814453125, -0.048492431640625, 1.46484375, 0.7490234375, -0.6201171875, 0.40087890625, 0.1331787109375, -0.72705078125, -0.405517578125, -0.50439453125, -0.21337890625, 0.25341796875, -1.7001953125, -0.65576171875, -0.7314453125, -0.62890625, -0.382080078125, 0.68408203125, -2.16796875, -1.3134765625, 1.1064453125, -0.416015625, -0.61865234375, -1.5966796875, 0.4150390625, -0.19677734375, -0.66357421875, -0.316650390625, 0.6318359375, -0.4794921875, -0.1239013671875, -1.0263671875, 1.0654296875, -0.1414794921875, 0.333251953125, 0.11468505859375, -0.297607421875, 0.2685546875, 0.59912109375, 0.38232421875, 0.9775390625, -0.163818359375, 1.609375, -0.73095703125, -0.57080078125, 0.339599609375, 0.4658203125, -0.7744140625, 0.032867431640625, 0.103515625, -0.3935546875, 1.1875, -1.0625, -0.2724609375, -0.39599609375, 0.25927734375, 0.0303497314453125, 1.591796875, -0.457275390625, 0.482421875, 0.427001953125, 0.0672607421875, 1.8076171875, 0.2279052734375, 0.7724609375, 0.90576171875, -0.135498046875, 0.155029296875, -0.30419921875, -2.015625, -0.42138671875, -0.0301513671875, 0.55908203125, 1.0009765625, 0.1085205078125, -1.28515625, -2.0234375, -0.203125, -0.3662109375, -1.36328125, -1.0478515625, 0.90625, 0.415283203125, 0.06982421875, 0.37109375, -0.31689453125, 0.2384033203125, -1.037109375, -1.3583984375, -0.84130859375, -0.327880859375, 0.94287109375, 0.9609375, -1.8447265625, 0.242919921875, -0.03875732421875, -0.469970703125, -1.283203125, -0.4208984375, 0.10205078125, 0.8486328125, 1.0966796875, 0.2861328125, 0.60986328125, 0.35400390625, 1.35546875, -0.8916015625, 0.56787109375, 0.6533203125, 0.509765625, 0.293212890625, -1.154296875, 1.3173828125, 0.42919921875, -1.3759765625, 0.433349609375, -0.11859130859375, 0.33203125, 0.435791015625, -1.3603515625, 0.74169921875, 2.095703125, -0.28662109375, 0.012481689453125, 0.3203125, 0.78759765625, -0.6162109375, -0.5439453125, -0.339111328125, 2.337890625, -0.34765625, 2.953125, -0.3359375, -1.24609375, -1.65234375, -0.10675048828125, 0.7548828125, -0.27734375, -0.9140625, -0.93994140625, 0.68017578125, -0.2288818359375, 0.53515625, 1.1884765625, 0.54443359375, 2.11328125, -0.486572265625, -0.63671875, 0.281982421875, -1.0439453125, 1.783203125, -0.41015625, -0.79052734375, -0.2408447265625, -0.78857421875, -0.5234375, 1.1328125, 0.564453125, -0.34912109375, 0.59716796875, 0.33203125, 0.2529296875, -0.59423828125, 0.2249755859375, -1.2119140625, -0.38818359375, 0.8720703125, 0.0380859375, -0.4111328125, 0.44873046875, -1.0546875, 0.916015625, -0.814453125, 0.08062744140625, -0.486328125, 0.2060546875, -1.0244140625, -0.04559326171875, 0.212646484375, -0.463134765625, -0.5517578125, -0.032135009765625, -0.54638671875, -0.6611328125, 1.376953125, -0.7158203125, -1.39453125, 0.77197265625, -0.6181640625, 0.027984619140625, -1.1123046875, 0.11474609375, 0.333740234375, -2.39453125, -0.2734375, -0.40966796875, -0.9609375, 1.0166015625, -2.216796875, -0.8681640625, 0.0875244140625, -0.1221923828125, 1.0791015625, -0.5068359375, 0.0799560546875, 0.7060546875, -0.06561279296875, 1.2236328125, -0.1341552734375, -0.68603515625, 0.373291015625, 0.02984619140625, 0.81640625, -0.41748046875, 1.755859375, -0.72509765625, -0.26171875, -1.1943359375, -2.068359375, -0.669921875, -1.0087890625, 0.724609375, 0.74755859375, -1.0302734375, -0.1483154296875, 0.315185546875, -0.7822265625, 0.580078125, -0.365478515625, 0.428466796875, -0.427978515625, -1.2216796875, 1.169921875, 0.051177978515625, 0.1929931640625, 1.6982421875, 0.380126953125, 0.5869140625, -1.076171875, -0.365966796875, -0.9501953125, 0.11474609375, 0.0133514404296875, 0.88720703125, -0.9697265625, 0.01666259765625, -0.59912109375, -0.64208984375, -0.34375, -0.28857421875, -1.2666015625, 1.0107421875, 0.99853515625, -2.193359375, -0.7314453125, -1.5244140625, 1.869140625, 1.171875, -0.32666015625, -0.8828125, 1.341796875, 0.6357421875, 0.39697265625, 1.453125, 0.2861328125, 1.0498046875, -0.072021484375, 0.006988525390625, 0.52392578125, 0.97607421875, 1.26171875, -0.1319580078125, -0.5791015625, -0.35546875, -0.63818359375, 1.06640625, 1.0419921875, 0.5361328125, 0.50634765625, 0.94189453125, 1.9658203125, -0.022430419921875, 0.9404296875, -0.53173828125, 0.02972412109375, 0.364501953125, 0.00555419921875, 2.45703125, -0.329345703125, -0.8955078125, -0.8994140625, -0.041290283203125, 0.5732421875, -0.68359375, -0.998046875, 1.2392578125, 1.15625, 1.9669532775878906e-06, 0.73828125, 0.4638671875, -0.44873046875, 0.0298309326171875, 1.2138671875, 0.23583984375, 0.2237548828125, 0.11932373046875, 0.08587646484375, 0.01430511474609375, -1.017578125, 0.078369140625, 0.90625, -0.65380859375, 0.76513671875, -0.08831787109375, 0.787109375, 1.095703125, -0.2034912109375, 0.39111328125, -0.724609375, -0.373291015625, -0.306640625, 1.3896484375, -1.0732421875, -1.859375, -1.486328125, -0.1197509765625, -0.99462890625, 2.0, 0.1826171875, 0.10101318359375, -1.791015625, -0.2493896484375, 0.431884765625, 0.277099609375, -2.005859375, -1.3681640625, -0.183837890625, 1.2177734375, -0.0225982666015625, 0.3935546875, -0.382568359375, 0.1907958984375, -0.37646484375, 0.642578125, 0.3076171875, -0.11822509765625, -0.2218017578125, 1.8251953125, 1.119140625, 0.73681640625, 0.86474609375, -1.201171875, 0.2109375, 0.1748046875, 0.299560546875, 0.15478515625, -1.1123046875, -0.70947265625, 0.491943359375, -1.447265625, -1.8076171875, 1.5380859375, 0.3779296875, 0.5009765625, -0.634765625, 1.5302734375, 0.1688232421875, -0.1103515625, 0.034423828125, -0.172607421875, -0.134521484375, -0.76416015625, -1.1533203125, -0.241455078125, 0.93701171875, 1.2763671875, -0.40087890625, -0.888671875, -1.55078125, -0.27783203125, 0.70947265625, -0.344482421875, -0.09295654296875, 1.9599609375, 0.65869140625, 0.78466796875, 0.88330078125, -1.037109375, 1.1103515625, -1.642578125, 0.28564453125, 0.321533203125, 0.1343994140625, 1.0517578125, 0.435791015625, -0.417236328125, 0.02593994140625, 1.064453125, 0.1634521484375, -1.1630859375, 0.340576171875, -1.90625, -0.08538818359375, 2.20703125, 1.103515625, 0.42236328125, -1.9091796875, -2.25390625, -0.109375, 0.6640625, 1.38671875, -0.2021484375, 0.61962890625, 0.148681640625, -2.298828125, 1.283203125, -1.048828125, 0.177490234375, 0.51220703125, -0.11749267578125, -1.9521484375, 1.08984375, -0.5927734375, 0.05584716796875, -0.94287109375, -0.49169921875, 1.25390625, -0.958984375, 1.486328125, -0.328369140625, 0.1514892578125, 1.0673828125, -0.7958984375, 1.419921875, 1.3017578125, -0.38134765625, 0.3583984375, -0.447998046875, 1.595703125, 0.708984375, -0.734375, -1.42578125, -0.1627197265625, 0.5859375, -1.03125, 0.418212890625, -0.365478515625, -0.44921875, -0.24169921875, -0.3388671875, -0.78857421875, -0.422119140625, -0.71435546875, -0.43359375, -0.79736328125, -0.361572265625, 0.599609375, 0.78759765625, 1.197265625, 0.72216796875, 0.5390625, -0.0982666015625, -0.1981201171875, 1.4541015625, -0.55859375, -0.362060546875, 1.279296875, -0.533203125, -0.09088134765625, -0.669921875, -0.1365966796875, 0.2120361328125, 0.405517578125, 1.662109375, -0.74658203125, -0.775390625, -1.0361328125, 0.035858154296875, 0.86474609375, -0.22802734375, 0.334716796875, -1.2578125, 0.250732421875, -1.369140625, -0.9482421875, -0.52197265625, 1.5009765625, 1.478515625, 0.92578125, -0.58837890625, -1.873046875, 1.7978515625, -0.57275390625, 0.99462890625, 0.38916015625, 1.154296875, 2.810546875, 1.5361328125, 0.66748046875, 0.72021484375, -1.1142578125, 1.35546875, -0.201416015625, 0.46875, 0.6708984375, 0.640625, 1.3828125, 1.0576171875, 0.73291015625, 1.3837890625, 1.9541015625, 0.431884765625, 0.484619140625, 0.15283203125, -1.3369140625, 0.50244140625, 1.8037109375, -0.2734375, -0.085205078125, 1.1640625, -0.01922607421875, 0.51220703125, 0.6396484375, 0.88232421875, -0.20458984375, -1.125, 1.541015625, -1.0654296875, 0.369140625, -0.6943359375, 0.30322265625, 0.9609375, 1.4677734375, 0.65185546875, 0.96826171875, -1.423828125, -0.54541015625, -0.312255859375, 0.900390625, -0.381591796875, 0.2470703125, -2.546875, 0.28466796875, -0.455078125, -1.2587890625, 0.65966796875, -0.37890625, 0.97265625, -0.443603515625, 0.35888671875, -0.76416015625, -0.4609375, 0.28369140625, -0.30859375, -0.343505859375, -1.2001953125, -0.1591796875, 1.7275390625, -0.01168060302734375, 0.08868408203125, -0.61767578125, 0.83984375, -0.68212890625, -0.76904296875, 0.32861328125, -1.0869140625, 0.281005859375, 0.2252197265625, -0.18408203125, -0.5537109375, -0.0721435546875, 1.1123046875, -0.87158203125, 2.033203125, -0.04351806640625, 1.2490234375, 0.196533203125, -1.9267578125, 0.67919921875, 0.334716796875, -0.85400390625, 0.5380859375, 1.4404296875, 1.171875, -0.94140625, 0.78564453125, 1.62109375, 1.60546875, -0.71875, 0.07757568359375, -0.51708984375, -0.7548828125, 0.640625, 0.474609375, -0.151611328125, 1.5009765625, -0.79150390625, 0.76611328125, 1.513671875, 0.59716796875, 0.1513671875, -1.93359375, 0.0838623046875, 0.233642578125, 1.37890625, 0.0266876220703125, 0.73828125, 0.44921875, 1.4111328125, -1.841796875, -0.304443359375, -0.70263671875, -0.053558349609375, 0.287841796875, 0.4521484375, -0.307373046875, 1.0791015625, 1.064453125, 0.39306640625, -0.305419921875, -0.056427001953125, 1.1640625, -1.328125, 0.85595703125, -0.4638671875, -0.405029296875, -1.251953125, 0.78662109375, -0.321533203125, 0.23681640625, -0.880859375, -1.5830078125, 1.0625, 0.5625, -0.335205078125, -1.0263671875, -1.98046875, -0.3828125, 0.7978515625, -0.58740234375, 0.06842041015625, -0.364013671875, -0.81689453125, -2.1640625, -0.463623046875, 2.837890625, 0.1893310546875, 0.24658203125, 0.1666259765625, -0.087890625, 0.1563720703125, -0.17236328125, -1.470703125, 1.318359375, 0.9951171875, -0.399169921875, -1.6240234375, -0.92529296875, 0.6708984375, 0.0823974609375, -1.908203125, -0.27685546875, 0.10406494140625, 0.361083984375, -2.1640625, -0.65966796875, -0.437744140625, 0.5361328125, 0.63134765625, -0.89892578125, 0.90869140625, -1.2705078125, -0.5908203125, -1.0302734375, 1.2353515625, 1.8369140625, -0.1470947265625, 0.0273895263671875, -1.685546875, 0.9912109375, 1.78515625, -0.85302734375, -1.2890625, 0.2154541015625, -0.69287109375, 1.5263671875, 1.5625, -0.05841064453125, -0.3662109375, 0.8193359375, -0.52392578125, 0.46533203125, -0.5244140625, -0.62060546875, 0.33544921875, -0.0648193359375, 1.0791015625, -0.85302734375, 0.443359375, 0.29296875, 1.6875, -1.0634765625, 0.5439453125, 0.71923828125, 0.376220703125, 1.095703125, -0.93701171875, 0.8642578125, 0.437744140625, 0.29736328125, -0.361572265625, 0.04217529296875, 2.3125, -1.9287109375, -2.150390625, -0.994140625, -0.29736328125, 0.363525390625, -0.9599609375, -0.7158203125, 1.68359375, 1.021484375, -1.48828125, 3.40625, -1.76171875, -2.267578125, -0.09112548828125, -1.37890625, -1.8271484375, -0.7783203125, 0.29638671875, -0.83544921875, 0.055938720703125, -0.572265625, 1.2353515625, -0.1195068359375, 0.43212890625, 0.82421875, -0.546875, 0.0799560546875, -0.2142333984375, -2.337890625, -0.08880615234375, 0.052642822265625, -1.2275390625, 1.5810546875, 0.013641357421875, -0.007404327392578125, 0.469970703125, 1.0615234375, 0.207763671875, 0.7939453125, -0.5703125, 0.197021484375, 0.927734375, 1.4892578125, -0.022216796875, -0.069580078125, 1.5654296875, 0.65576171875, -0.4404296875, 0.011199951171875, -0.65625, -0.029205322265625, 1.05859375, 1.478515625, 1.1611328125, -0.430908203125, -0.353515625, 0.0634765625, 1.451171875, -0.57861328125, -0.87451171875, 1.625, 0.4814453125, 1.0888671875, -0.845703125, 1.2060546875, -1.0703125, -0.3125, -1.3984375, -1.333984375, -0.6689453125, 0.818359375, -0.44580078125, 0.50830078125, -0.2900390625, 1.7890625, -0.1707763671875, 0.3330078125, 1.7119140625, 0.15625, -0.5498046875, 0.3203125, -0.11749267578125, 1.470703125, 0.9521484375, -0.50439453125, 1.4169921875, -0.409912109375, 0.0261688232421875, 1.09375, 0.468017578125, -0.09619140625, -0.58642578125, 0.88671875, -0.1348876953125, 0.55126953125, 0.58349609375, -0.54296875, -1.4912109375, -0.1793212890625, -1.673828125, -0.0185394287109375, -2.44921875, -0.81787109375, -1.548828125, -1.5048828125, -0.96826171875, 0.351806640625, 0.4033203125, 0.59912109375, -0.80322265625, -0.367431640625, 0.69970703125, -1.6240234375, 1.337890625, 0.498779296875, 0.95458984375, -2.384765625, 1.306640625, -0.75244140625, 0.86572265625, 0.391845703125, -1.21875, -0.96728515625, 0.123291015625, 0.09765625, 1.0830078125, 0.057342529296875, -0.450927734375, -2.60546875, 0.2271728515625, 0.619140625, -0.06695556640625, -0.59619140625, -0.947265625, 0.77783203125, 1.548828125, 0.2919921875, -1.2099609375, 0.63330078125, -0.2100830078125, -0.58984375, 1.109375, -1.8173828125, -0.68310546875, -0.274169921875, -0.56005859375, 1.3232421875, -0.87353515625, -1.720703125, -0.88525390625, -1.0126953125, -0.484130859375, 0.7138671875, -0.759765625, 0.50732421875, 0.77685546875, 1.4404296875, -0.65625, -0.0794677734375, -1.8076171875, -0.29443359375, -1.021484375, -1.720703125, -0.740234375, 1.66015625, 0.587890625, -0.434814453125, -1.33203125, -0.1875, -0.284912109375, 0.66455078125, 0.391845703125, -0.892578125, 1.00390625, 0.08978271484375, 0.2144775390625, -0.52392578125, 0.359375, -1.2333984375, -0.51611328125, 0.1590576171875, -0.06787109375, -1.0498046875, -1.39453125, -1.974609375, 1.2197265625, 1.3701171875, 0.331298828125, 0.49609375, -0.021575927734375, -0.62548828125, 0.69677734375, 0.126953125, 1.6123046875, -1.5908203125, -1.078125, 0.59619140625, 1.609375, 0.5302734375, -0.64697265625, 0.06329345703125, -0.0472412109375, -0.2481689453125, -2.03515625, -0.22900390625, 1.3330078125, 1.7275390625, 0.923828125, 0.92529296875, -0.2198486328125, 0.1805419921875, 0.8408203125, 0.398681640625, 0.75732421875, -1.595703125, 1.0146484375, -0.087646484375, -0.1383056640625, 1.287109375, 0.30322265625, -1.8701171875, -0.45654296875, -0.50830078125, -0.771484375, -1.0322265625, 0.888671875, -1.1279296875, -0.01166534423828125, -1.9873046875, -1.677734375, -1.6396484375, -0.42138671875, 1.6748046875, -0.06982421875, 1.587890625, -0.69970703125, 1.1591796875, 0.431396484375, 0.05615234375, 0.8408203125, -0.1693115234375, 0.08953857421875, -1.0341796875, -1.6123046875, 0.94091796875, 0.270751953125, -1.037109375, 0.37109375, 1.033203125, -0.405029296875, -0.86572265625, 0.73974609375, 0.376708984375, 0.060272216796875, -1.1826171875, -1.4140625, -0.07952880859375, 0.073486328125, -0.673828125, -1.7724609375, -0.8876953125, -0.1390380859375, 0.2919921875, 0.1236572265625, 0.55078125, 0.828125, -1.9833984375, -0.43017578125, -0.127197265625, 0.387451171875, -0.8974609375, 0.1739501953125, -1.171875, 0.30029296875, 1.6708984375, 0.1083984375, 0.5927734375, -0.578125, -0.1673583984375, 0.66845703125, 0.857421875, 0.5302734375, -0.10638427734375, -0.73486328125, 0.13037109375, -3.333984375, -0.5771484375, -1.1005859375, -0.77783203125, 1.5244140625, -0.52099609375, -0.26513671875, -0.53076171875, 0.654296875, 0.5498046875, -0.236083984375, 0.284912109375, 0.076416015625, 1.021484375, -1.2880859375, 0.638671875, 0.1912841796875, 0.6337890625, -1.0830078125, -1.7001953125, 0.76220703125, 0.6337890625, 0.96142578125, -2.048828125, -1.431640625, -1.0224609375, -1.982421875, -1.3017578125, -1.0361328125, -0.82421875, -0.7724609375, -1.26171875, -0.9970703125, -1.1396484375, -0.431396484375, 0.7158203125, 0.466552734375, -0.4296875, -0.430908203125, -0.5634765625, -0.0206756591796875, -1.8876953125, 0.225830078125, 1.6650390625, -0.468505859375, 0.62890625, 0.369384765625, 1.0224609375, 0.1448974609375, 0.529296875, 0.251953125, 0.87353515625, -0.12115478515625, 1.33203125, 1.375, -1.0341796875, -0.359130859375, -0.93505859375, 0.99658203125, 1.72265625, -0.80810546875, 0.974609375, -0.75732421875, 0.71875, 0.71044921875, -0.71875, 0.69091796875, -1.2314453125, -1.15625, 0.8955078125, -0.05096435546875, 0.892578125, 0.83056640625, 0.07257080078125, 1.111328125, 2.400390625, -1.98046875, 0.99658203125, 1.3603515625, 0.72998046875, 2.228515625, 0.900390625, -0.127197265625, -0.041229248046875, -0.403564453125, -0.54150390625, -1.71875, -1.6005859375, 0.7666015625, -1.48046875, 1.2470703125, -1.310546875, 0.41650390625, 0.232177734375, -1.4853515625, 1.822265625, 0.99609375, -0.163818359375, -0.3310546875, -0.450439453125, -0.716796875, 0.365478515625, -0.08990478515625, 1.8046875, 1.8681640625, -2.595703125, -1.892578125, 0.372314453125, -0.95068359375, 0.08465576171875, 1.6796875, -0.2354736328125, 0.007358551025390625, -0.244384765625, 2.013671875, -0.8515625, -0.765625, 0.59375, -0.453125, 0.90380859375, 1.3173828125, -0.5048828125, -0.60302734375, 0.2734375, -0.642578125, 1.3857421875, 0.80078125, -0.81494140625, -0.5400390625, 1.0009765625, -0.546875, -0.460205078125, 0.78369140625, 0.90771484375, -1.373046875, 0.412353515625, 0.07989501953125, -0.1639404296875, 0.810546875, 1.75390625, 0.314208984375, -2.005859375, 0.358642578125, -1.01171875, 1.6611328125, 0.88671875, -1.275390625, 1.5205078125, -0.5576171875, -1.001953125, -1.0205078125, 0.2230224609375, 1.3134765625, -0.47509765625, -1.8798828125, 0.3671875, -0.2193603515625, 0.86865234375, 0.759765625, -1.1328125, -0.46240234375, 0.371337890625, -1.29296875, 0.3759765625, 0.047332763671875, 0.79345703125, -0.2066650390625, -0.51611328125, 0.11468505859375, 0.06829833984375, 0.7734375, -0.29150390625, -0.2354736328125, 0.64599609375, -0.65625, -1.1865234375, 0.46484375, 1.5673828125, 0.352294921875, 0.951171875, 1.123046875, -0.89892578125, 0.271484375, -0.01678466796875, 0.050872802734375, 1.05859375, -0.3740234375, -1.4208984375, -0.5634765625, 0.51416015625, 0.88818359375, 0.5849609375, 0.339111328125, -2.232421875, -1.35546875, -1.880859375, -0.962890625, -1.5224609375, -1.2900390625, 0.3125, 1.26953125, 0.314697265625, -0.54345703125, -0.06451416015625, 0.42822265625, 0.77880859375, -0.46044921875, 0.07232666015625, -0.697265625, -1.0615234375, 0.67919921875, 0.192138671875, -0.02630615234375, 0.2900390625, 0.5634765625, -0.2261962890625, -0.5322265625, 1.458984375, -1.3544921875, -0.85400390625, 1.2958984375, -1.87890625, 0.393798828125, 1.21484375, -0.5341796875, 1.4248046875, 0.4423828125, -0.75732421875, 0.08892822265625, -0.3359375, 0.1622314453125, 0.5126953125, 1.9306640625, -1.443359375, 0.0262603759765625, 0.634765625, -0.096923828125, -0.5126953125, 1.658203125, -0.005573272705078125, 0.56591796875, 0.8466796875, -0.9970703125, 1.5966796875, 0.499755859375, 0.2354736328125, -0.607421875, 0.19677734375, -0.1602783203125, 0.72265625, -0.291748046875, 0.67041015625, -0.96923828125, 1.7109375, 1.1171875, 0.83349609375, -0.61474609375, -1.6845703125, 0.5, -0.94775390625, -0.6376953125, 0.332763671875, -0.52734375, -0.82568359375, 0.69970703125, -1.3642578125, 0.39013671875, -0.54443359375, 1.9189453125, -0.5517578125, 0.185302734375, 0.32080078125, 0.260009765625, -0.68408203125, 0.00457763671875, 1.107421875, 0.62109375, -0.18115234375, 0.9619140625, 0.7529296875, 1.203125, 0.5166015625, -0.466796875, 0.63037109375, -0.888671875, 0.98681640625, -0.76904296875, 0.364501953125, -0.4501953125, 0.57373046875, -0.23388671875, 0.07684326171875, 0.62255859375, -0.70458984375, 0.130615234375, 0.595703125, 0.5068359375, -0.300048828125, 1.6318359375, 0.65673828125, 0.286376953125, -0.45361328125, 0.407958984375, 0.2332763671875, -0.038909912109375, -0.5791015625, 0.492431640625, -1.1943359375, -0.39208984375, 1.013671875, 0.0740966796875, 0.475341796875, 0.1217041015625, -0.7353515625, 0.1478271484375, -1.095703125, -0.10382080078125, -0.00576019287109375, -0.49365234375, 0.34033203125, 0.102783203125, -0.06396484375, -2.287109375, 0.41357421875, -0.06793212890625, -1.5263671875, 1.482421875, -0.80419921875, -1.1953125, 1.41015625, -0.318115234375, 0.487548828125, -0.406005859375, -1.1123046875, -1.0615234375, -0.1212158203125, -0.615234375, 0.61328125, 0.1822509765625, 0.365234375, -0.426025390625, 0.72900390625, -0.336669921875, 0.87548828125, 0.4150390625, 0.6494140625, 0.87744140625, -0.05841064453125, -0.4462890625, -0.662109375, 0.28759765625, 1.71484375, -0.12396240234375, -1.5302734375, -2.05859375, -0.4794921875, -1.017578125, -0.81884765625, 0.98486328125, -0.59765625, -1.5380859375, -0.5703125, 1.1318359375, -0.41455078125, 0.363037109375, 1.9189453125, -1.0625, -1.0224609375, 0.180419921875, -1.4423828125, 0.6494140625, 0.7216796875, -0.42822265625, -0.378173828125, -0.403564453125, 1.0283203125, -1.37890625, 0.379638671875, 0.693359375, -0.716796875, 1.5380859375, 0.7705078125, 0.8544921875, -1.505859375, 0.2059326171875, 1.583984375, -1.732421875, 0.68994140625, 0.195068359375, 0.908203125, -1.365234375, -0.1943359375, 0.51171875, -2.51171875, -0.90966796875, -3.080078125, 0.4013671875, 0.86962890625, 1.1376953125, -0.005329132080078125, 1.0693359375, -2.201171875, -0.0845947265625, -0.00957489013671875, -0.9931640625, 0.7060546875, -2.578125, -2.60546875, 0.353759765625, -0.1361083984375, 1.5546875, -0.5380859375, 0.2235107421875, 1.0712890625, -0.247314453125, -1.423828125, 0.05145263671875, -0.927734375, -0.66015625, -0.1434326171875, -0.0687255859375, -0.62841796875, 1.826171875, 0.423828125, -1.3935546875, 0.6904296875, 0.505859375, -0.60302734375, -1.3720703125, 1.005859375, -0.97607421875, -1.1513671875, 0.0833740234375, 1.076171875, -0.63623046875, -0.6796875, -2.1796875, 0.810546875, -0.64599609375, -0.2744140625, 0.6669921875, 0.326416015625, 0.1485595703125, -1.0224609375, 0.66552734375, -1.041015625, 0.8046875, 0.6396484375, -0.509765625, 1.2275390625, -0.0295867919921875, -0.75390625, -0.46435546875, -0.09857177734375, -0.61962890625, 0.29443359375, 0.53515625, -0.69580078125, -1.6376953125, 2.46875, 0.8818359375, -0.6171875, -1.330078125, 2.23828125, -0.9033203125, 0.311279296875, 0.62890625, 1.2783203125, 0.82861328125, -1.947265625, -1.490234375, 0.44921875, 0.391845703125, -1.82421875, 0.498291015625, 0.097412109375, 0.332275390625, 0.1123046875, -0.833984375, -1.0224609375, 1.2158203125, -0.6533203125, 1.2607421875, -0.437744140625, -0.6435546875, -0.2132568359375, -1.275390625, -1.5712890625, 1.1865234375, 0.80322265625, -0.7294921875, -0.98974609375, -0.85205078125, 1.3984375, 0.2498779296875, 0.712890625, -0.1217041015625, -0.78955078125, -0.20947265625, 0.83203125, -0.60595703125, -1.509765625, -0.07855224609375, 1.18359375, -0.736328125, 0.68896484375, 0.10052490234375, 0.1163330078125, 1.2958984375, 0.92333984375, 0.80810546875, -0.091552734375, -0.69580078125, -1.48046875, 0.27685546875, -0.5830078125, -0.67138671875, 0.127197265625, -0.9345703125, 0.161376953125, -0.1318359375, 2.2421875, -0.98486328125, 0.2418212890625, -2.279296875, 0.9111328125, 0.82958984375, -0.2049560546875, -2.015625, 0.034088134765625, 0.326416015625, 0.2381591796875, -0.493896484375, -2.205078125, -1.0146484375, -0.9306640625, 0.53857421875, -1.001953125, 0.0419921875, 0.0157928466796875, -1.119140625, 0.452880859375, -0.08489990234375, 0.051849365234375, 0.7890625, 0.045135498046875, 0.58984375, 0.18701171875, 0.07666015625, 0.34375, -1.16796875, -1.9404296875, 0.1806640625, -0.312744140625, 0.38037109375, 0.447021484375, 1.5634765625, -0.322509765625, 0.332275390625, -2.19921875, 1.078125, 1.376953125, -0.06610107421875, -0.849609375, 0.0816650390625, 0.37353515625, 0.1953125, -0.094482421875, -0.46728515625, -2.537109375, 0.50390625, 0.576171875, -0.4482421875, 2.572265625, -0.08251953125, -0.39501953125, -0.73193359375, 0.599609375, -1.234375, 1.2890625, -0.175537109375, -1.259765625, 0.59130859375, 0.051513671875, 1.146484375, 0.9287109375, 0.262939453125, 0.3330078125, -1.169921875, -0.14892578125, -0.105712890625, -0.51708984375, -0.0161590576171875, 0.1806640625, 1.0234375, 0.169189453125, -1.4375, -0.64404296875, 0.90087890625, 0.11444091796875, 1.4853515625, -1.1552734375, 0.2322998046875, -0.08074951171875, 0.042510986328125, -0.09588623046875, 0.80859375, 2.068359375, 0.71044921875, -1.9443359375, -0.7919921875, 0.6845703125, 0.24267578125, -0.75, -0.6279296875, -2.03515625, -0.8505859375, 0.165283203125, 0.63623046875, -0.51953125, -1.2177734375, -1.6376953125, 1.5966796875, 1.4541015625, 1.81640625, -0.88916015625, 0.10198974609375, -1.03515625, 0.456787109375, -0.3896484375, 2.4375, -0.420654296875, 0.552734375, 1.265625, 2.150390625, -0.91650390625, -1.236328125, 0.72509765625, 0.73193359375, 1.734375, 1.388671875, -2.451171875, -0.1705322265625, 1.91796875, 0.78955078125, -1.5302734375, 1.791015625, -1.6552734375, -1.89453125, -0.55615234375, -0.609375, -0.982421875, 0.6044921875, -1.4384765625, -1.052734375, -0.279052734375, -1.2080078125, -0.279296875, -1.099609375, 0.2293701171875, -1.724609375, -0.56005859375, -0.32470703125, 1.5263671875, -0.6279296875, 0.587890625, -2.203125, -1.791015625, 0.451416015625, -0.75048828125, -0.39697265625, 0.34716796875, 0.60107421875, -0.87939453125, -0.285888671875, 0.470947265625, 0.517578125, -0.71630859375, 0.99072265625, -0.64697265625, 0.708984375, -1.0517578125, -1.232421875, -0.89208984375, 0.1619873046875, 0.970703125, 0.455810546875, -0.027740478515625, 0.58251953125, -0.435546875, -0.25, -1.1396484375, -0.82275390625, -0.2724609375, -0.057525634765625, 0.6650390625, 1.2744140625, -0.56494140625, 2.091796875, 0.006412506103515625, 1.4931640625, 0.30810546875, 0.64208984375, -0.1072998046875, 0.22216796875, 0.5625, 0.345703125, 0.177734375, -0.04315185546875, -1.314453125, 1.3388671875, 1.513671875, -0.20947265625, -0.294189453125, 0.77734375, -1.50390625, 1.4423828125, 0.46337890625, -0.35693359375, -0.310546875, -0.84814453125, -0.94580078125, 0.82470703125, -0.359619140625, 0.60400390625, -0.98779296875, -0.2235107421875, 0.362060546875, 0.006565093994140625, 0.491455078125, -2.30078125, -0.80517578125, -0.8935546875, 0.6298828125, -1.0107421875, -0.66064453125, 1.1806640625, 0.7548828125, -0.143310546875, 0.17529296875, 0.740234375, -0.461181640625, 1.462890625, -0.8037109375, 1.28515625, -0.1524658203125, 0.24462890625, -0.048431396484375, -0.168701171875, -1.4638671875, 0.5703125, -0.1502685546875, 0.9140625, -0.60986328125, -0.30859375, 1.1884765625, -0.73486328125, 1.3564453125, 1.244140625, 0.23388671875, 0.266845703125, 0.253173828125, -1.703125, -0.83154296875, -0.92138671875, -0.98779296875, 0.0290679931640625, -0.01375579833984375, -0.73974609375, -0.89453125, -1.9658203125, 0.072509765625, 1.833984375, 0.1651611328125, -1.509765625, -0.95166015625, -0.61669921875, -1.068359375, 0.78759765625, -0.2193603515625, 1.45703125, 0.80419921875, 1.1591796875, 1.208984375, 0.2193603515625, 1.4990234375, 0.058502197265625, 1.109375, 0.415771484375, -0.318115234375, -0.261962890625, -0.441162109375, 0.1787109375, 0.72216796875, -0.5458984375, -1.123046875, 0.6474609375, 0.77490234375, -0.90087890625, -0.067138671875, -2.029296875, -0.031707763671875, -0.5205078125, 0.896484375, -0.3466796875, 0.5009765625, -0.42529296875, -1.23046875, 0.7431640625, -0.80859375, 0.8759765625, -0.243408203125, -0.20947265625, -0.0811767578125, -0.3974609375, 1.2119140625, -0.63916015625, 1.00390625, -0.69873046875, 1.6904296875, 0.25341796875, -0.4453125, 1.1650390625, -0.490234375, 0.02520751953125, -0.277099609375, -1.646484375, -1.34375, 2.052734375, 0.08782958984375, 1.1083984375, 1.39453125, 0.60302734375, 0.8955078125, -1.806640625, 0.55517578125, 0.18603515625, 0.873046875, -0.312744140625, -0.263916015625, 0.0810546875, 0.58056640625, 0.19873046875, 0.481689453125, 0.93359375, 0.91162109375, 0.478759765625, 1.3779296875, -0.048095703125, -1.7255859375, -0.467041015625, 0.9501953125, -1.1181640625, 0.78271484375, -0.031646728515625, 0.62548828125, -0.161865234375, 0.95751953125, -1.1669921875, -0.339111328125, 0.344482421875, 0.412353515625, 0.8251953125, 0.59619140625, -0.76171875, -1.845703125, -0.34326171875, -0.9912109375, -0.52001953125, 0.76171875, 1.1005859375, -0.1937255859375, -1.1884765625, -0.83984375, 0.552734375, -0.462158203125, 0.88916015625, 0.61474609375, 0.83349609375, -0.87548828125, 0.1416015625, -0.68212890625, 0.10308837890625, 0.0004374980926513672, -0.06378173828125, 0.62158203125, -0.8388671875, -0.379638671875, 2.5546875, 0.72998046875, 0.1331787109375, -0.6767578125, 0.3603515625, -0.64599609375, -0.46142578125, -0.2431640625, -0.064453125, 1.41015625, -0.0885009765625, 1.1767578125, -0.59814453125, 0.08380126953125, 0.57861328125, 0.50244140625, -0.146240234375, 0.08856201171875, -0.01568603515625, 0.50244140625, -0.1612548828125, 0.9560546875, 1.560546875, -0.6806640625, 0.50048828125, 0.72216796875, 0.6259765625, 1.9345703125, 0.7744140625, 1.041015625, -0.3076171875, 0.386962890625, 1.2001953125, -0.6376953125, -1.220703125, 1.912109375, 0.72900390625, 0.69970703125, -0.53125, 1.1201171875, -1.29296875, -1.0615234375, -2.541015625, -2.294921875, 0.323486328125, 1.3544921875, 1.5126953125, -1.2724609375, -0.63818359375, -0.68212890625, -0.1060791015625, -0.4921875, 0.6591796875, 0.398681640625, 1.5361328125, -0.30322265625, 0.88330078125, 0.61865234375, 1.2138671875, 1.0400390625, 1.3564453125, 2.5, 0.43359375, -0.220947265625, 0.53173828125, 1.0849609375, -0.765625, 0.58544921875, 0.48291015625, -0.994140625, -1.0126953125, -1.2265625, 1.341796875, 2.673828125, -0.0557861328125, 0.9931640625, -0.0183563232421875, 0.5615234375, 0.09576416015625, -0.91162109375, -0.97802734375, 0.07781982421875, 0.407470703125, -0.05511474609375, 0.69873046875, -0.51806640625, 0.55615234375, 0.306640625, -1.0703125, -1.044921875, 0.8310546875, -0.9091796875, 0.9892578125, 0.55224609375, 0.5107421875, -2.23828125, 0.7080078125, 0.76513671875, -1.5908203125, 1.220703125, -1.302734375, 0.103515625, 0.587890625, -1.0400390625, 0.6171875, -0.5078125, -0.52197265625, 1.033203125, -2.169921875, 0.2164306640625, 0.004581451416015625, -0.2191162109375, -0.6728515625, -2.126953125, -0.94921875, -1.1982421875, 0.9208984375, 0.615234375, 0.462158203125, 0.1300048828125, 0.68310546875, 0.7783203125, 0.40673828125, -0.0570068359375, 1.0947265625, -0.6708984375, 0.52001953125, 1.6953125, -0.07989501953125, 0.0877685546875, 0.84033203125, -0.052276611328125, -0.250244140625, -0.9765625, -1.068359375, 0.4736328125, -0.9296875, -0.478271484375, -0.55712890625, 1.111328125, 0.08642578125, 0.9482421875, 0.21923828125, -1.240234375, 0.09136962890625, -0.392578125, -0.2298583984375, -1.1962890625, -0.2381591796875, -0.59619140625, 0.93310546875, 0.96728515625, 1.9619140625, -0.58154296875, -0.0105133056640625, -0.09246826171875, 0.11163330078125, -0.38134765625, 1.2880859375, 0.1986083984375, 1.5927734375, -1.1259765625, -0.7890625, 0.9921875, 2.154296875, 0.208251953125, 0.4697265625, 0.3056640625, 0.00650787353515625, 0.58740234375, 0.1446533203125, 1.375, 0.173583984375, 0.779296875, 0.93115234375, 2.66015625, 0.411865234375, 0.196044921875, 1.251953125, -2.1953125, 0.80615234375, 0.342041015625, 1.23046875, -0.67333984375, 1.3115234375, -0.1815185546875, -1.474609375, -0.316650390625, 1.15625, 0.669921875, -0.55224609375, 1.7216796875, 0.223388671875, -0.98388671875, 1.798828125, 0.73486328125, -2.296875, -0.07379150390625, 0.388916015625, -0.63525390625, -0.471923828125, -0.77001953125, 0.498291015625, -0.603515625, -1.8388671875, -0.5185546875, -0.6240234375, -1.0517578125, 1.0087890625, -0.57470703125, -1.4521484375, 0.087646484375, 0.95068359375, 1.33203125, 1.4765625, 0.456787109375, -1.3359375, 0.37939453125, 1.796875, -0.026153564453125, -1.2763671875, 0.482421875, 0.115234375, -1.2998046875, 1.2802734375, -1.07421875, 0.72119140625, 0.1422119140625, 1.5478515625, -0.1473388671875, -1.6650390625, -1.54296875, -0.345703125, -1.68359375, -0.7724609375, 0.173095703125, -1.25, -0.2003173828125, -0.2098388671875, -0.5517578125, -1.0546875, -0.51318359375, -1.0263671875, -0.07000732421875, -0.472412109375, 0.7119140625, 0.6044921875, 1.072265625, -0.007709503173828125, 0.104736328125, 0.50732421875, 1.0830078125, 1.751953125, 1.0390625, 0.202392578125, 1.5419921875, 0.2374267578125, -1.5556640625, -0.82421875, 0.09332275390625, -0.03289794921875, 1.4931640625, -0.275634765625, -0.2083740234375, 0.143798828125, 0.0794677734375, -0.09527587890625, -1.619140625, -1.521484375, 0.8359375, -0.1807861328125, 1.1806640625, 0.055877685546875, -0.59765625, 1.2041015625, 1.560546875, -0.4052734375, 1.328125, -0.352783203125, -0.90478515625, -0.480712890625, -1.1220703125, -0.76416015625, -0.2060546875, -0.94775390625, -1.0947265625, -0.12225341796875, -1.9140625, -0.6494140625, -0.41015625, -1.3896484375, -0.7568359375, -0.3291015625, 0.6962890625, -1.130859375, 0.51806640625, -1.4912109375, 0.487060546875, -1.322265625, 0.195556640625, 0.62548828125, -0.26611328125, -0.1236572265625, -1.302734375, -1.0341796875, -0.84716796875, 2.21875, -0.74951171875, 1.7216796875, 0.49267578125, -0.6376953125, -0.429443359375, -0.2257080078125, -0.59130859375, -0.31640625, -1.1005859375, -1.140625, 1.8642578125, -0.1259765625, -1.298828125, -1.8515625, -0.45751953125, 0.5419921875, 0.8828125, 0.37158203125, 2.109375, 1.7880859375, -0.0241851806640625, -0.44580078125, 0.08673095703125, 0.92138671875, 1.3427734375, -0.64306640625, -1.162109375, 0.093994140625, -0.1651611328125, 0.33935546875, -0.85986328125, 2.14453125, -0.11834716796875, -1.37109375, 0.83740234375, -0.7705078125, 1.79296875, -0.462890625, -1.6376953125, 0.73876953125, 0.87939453125, 1.0390625, -1.5546875, -2.740234375, 0.435302734375, 0.3203125, -0.853515625, -2.751953125, 1.642578125, 1.03515625, -0.0019969940185546875, -1.671875, -0.96826171875, 1.2880859375, 0.389404296875, 0.5263671875, 0.01361846923828125, -0.74560546875, -1.203125, 0.55517578125, 1.0712890625, 1.45703125, -0.0268096923828125, 0.204345703125, -0.053131103515625, -1.3603515625, -1.197265625, -0.13916015625, 0.394287109375, -1.05859375, -2.3828125, -0.08038330078125, -0.46630859375, -1.748046875, 2.287109375, 0.2088623046875, -1.0537109375, 0.0293426513671875, -0.4013671875, 0.43408203125, 0.4326171875, 0.460205078125, -0.12420654296875, -1.34375, 0.3056640625, 1.041015625, 0.88818359375, -0.31689453125, -1.859375, 1.1015625, -0.74560546875, 0.461669921875, -0.424560546875, 0.2449951171875, 0.7509765625, 0.6357421875, -0.419921875, -0.446533203125, 1.9462890625, 0.599609375, 1.478515625, 0.2822265625, -0.51416015625, -0.78369140625, 0.1336669921875, 0.6123046875, -0.042816162109375, -0.59130859375, -0.5341796875, -0.7490234375, -2.33203125, 1.1845703125, -0.076171875, 0.822265625, -2.142578125, -0.76806640625, -0.6845703125, 0.6787109375, -0.496337890625, -1.09765625, 0.270263671875, 1.0712890625, 0.43017578125, 0.81103515625, -0.52587890625, 0.247314453125, 0.347412109375, -0.399658203125, 0.90576171875, -0.257080078125, 0.54345703125, -1.4638671875, -0.306396484375, 0.473876953125, -0.23046875, -0.2412109375, -1.2666015625, -0.82177734375, -1.310546875, -1.1513671875, -0.3037109375, -2.03125, -0.44287109375, 1.568359375, -1.458984375, -0.0111846923828125, 0.41162109375, 0.8759765625, 0.79052734375, -0.7900390625, -0.381103515625, 0.994140625, -1.271484375, 0.019500732421875, 1.4833984375, 0.5390625, -1.0361328125, -0.689453125, 0.38623046875, -0.67724609375, -0.716796875, -1.0888671875, -1.7802734375, 0.60107421875, -1.7509765625, -0.09771728515625, 0.57275390625, -1.2958984375, 1.2265625, -0.79443359375, -0.221923828125, -2.681640625, 0.1287841796875, -1.1552734375, 0.1806640625, 1.166015625, -0.53564453125, 0.60693359375, 1.1494140625, 0.1300048828125, 1.15234375, -1.3759765625, 0.95263671875, -1.2041015625, -0.3505859375, 1.0849609375, 1.126953125, -1.1572265625, -0.258056640625, -0.398193359375, -1.5107421875, 0.6962890625, -0.685546875, 1.0458984375, -1.744140625, 0.277587890625, 0.90380859375, 0.07513427734375, 1.623046875, 0.189453125, 1.1435546875, -0.6689453125, -0.417236328125, 0.611328125, 0.449951171875, -0.857421875, -0.1722412109375, -0.73828125, 1.142578125, 0.419677734375, -1.970703125, -1.0302734375, -0.447265625, 1.2509765625, 0.95556640625, -0.65625, 1.2451171875, -0.63134765625, 0.079833984375, -0.841796875, -0.031402587890625, -0.4482421875, 0.416015625, -2.5390625, -0.1888427734375, -2.65234375, -1.6044921875, -0.458984375, -0.86083984375, 0.703125, -0.86669921875, -1.0537109375, 0.74853515625, -0.5439453125, -0.7021484375, -0.57177734375, 0.00701141357421875, -0.06524658203125, 0.73046875, 1.1953125, -0.2381591796875, -0.904296875, -0.85693359375, 0.564453125, 1.7041015625, 0.370361328125, -0.1044921875, 0.34326171875, 0.8125, 1.248046875, 0.52490234375, 0.35595703125, 0.56298828125, -0.406494140625, 0.409912109375, 0.000576019287109375, 0.24462890625, -0.52099609375, -0.63671875, 1.37890625, 0.409912109375, -0.418212890625, 0.03912353515625, 0.4228515625, -0.419189453125, 2.76171875, -1.1845703125, -2.328125, -1.3984375, 0.615234375, 0.31396484375, 1.8447265625, -1.8671875, -0.8046875, -0.263671875, -0.5712890625, -0.04791259765625, -0.548828125, -0.1712646484375, -0.09552001953125, 1.3740234375, 1.064453125, -0.208251953125, 1.5244140625, -0.1246337890625, -1.056640625, -0.03411865234375, -0.54345703125, 1.5888671875, -1.1611328125, -1.5048828125, 0.252197265625, 1.669921875, 2.951171875, 1.46875, -0.84375, 0.66259765625, 3.224609375, 0.73974609375, 0.44970703125, -1.0087890625, 0.8115234375, -0.331298828125, -0.11181640625, -0.71240234375, 0.039154052734375, 1.7275390625, 0.9365234375, -0.58349609375, -0.408203125, 0.239013671875, -1.5830078125, 0.029449462890625, 0.63232421875, -1.64453125, -0.0303192138671875, -1.654296875, -0.397216796875, 1.875, -0.52392578125, -0.73974609375, 0.139892578125, 0.50439453125, -0.30712890625, -1.18359375, -1.0146484375, 0.29833984375, -0.857421875, -0.041046142578125, 0.5732421875, -0.611328125, -0.30126953125, 0.60205078125, -2.26953125, -0.7998046875, -0.81591796875, -0.703125, -0.43603515625, -1.8681640625, 0.07928466796875, 0.79248046875, 0.326416015625, -0.149658203125, -0.5439453125, 0.08660888671875, -0.90087890625, -0.8671875, -0.78173828125, -0.81201171875, 0.904296875, -0.9736328125, -1.11328125, 0.330810546875, 0.2186279296875, 0.59130859375, -1.03515625, -1.7900390625, 0.517578125, -0.69580078125, -0.1737060546875, -1.3046875, -0.498291015625, -1.3369140625, 0.30224609375, 0.359619140625, -1.150390625, 0.08807373046875, 0.5048828125, 0.366943359375, 0.0224151611328125, -0.367919921875, -0.32568359375, -2.763671875, -1.626953125, -0.591796875, -0.287841796875, -0.505859375, -0.96630859375, 0.83203125, -0.36376953125, 1.4228515625, 1.1240234375, 0.45166015625, 1.041015625, -0.55908203125, 0.236083984375, -0.339111328125, 1.6591796875, 0.509765625, 2.55078125, 1.146484375, 1.291015625, -1.22265625, -1.419921875, -0.0318603515625, -1.8544921875, 0.7822265625, 0.607421875, 0.473388671875, 1.7861328125, 0.375732421875, 0.49169921875, -0.281982421875, 0.5107421875, -0.70556640625, 1.240234375, -0.27392578125, -1.5791015625, 0.241943359375, 0.0244903564453125, 1.380859375, 0.2091064453125, -0.158935546875, 0.212646484375, 0.147705078125, 0.07598876953125, -0.327392578125, -0.1634521484375, -0.2088623046875, -0.405517578125, -0.08843994140625, 0.66748046875, -0.46484375, -0.67041015625, 0.37255859375, 1.0400390625, -1.2880859375, -1.537109375, -0.208984375, -0.88232421875, 1.5478515625, 1.708984375, 1.6455078125, -1.0048828125, 0.0297393798828125, -0.359619140625, 1.185546875, 0.331787109375, -1.5927734375, 0.46728515625, -0.75, 0.25146484375, 0.01265716552734375, 0.12841796875, 0.51025390625, -0.1317138671875, 1.5498046875, 0.029541015625, 1.2001953125, 0.471435546875, 0.314453125, -1.708984375, -1.4462890625, 0.6337890625, -0.072265625, -0.85400390625, 0.6796875, -0.03875732421875, 1.076171875, -0.08636474609375, -1.8017578125, -0.12744140625, 0.6572265625, 0.180908203125, -1.728515625, 1.7119140625, 0.2384033203125, -0.06890869140625, -0.280029296875, 2.095703125, -0.3505859375, -0.76025390625, 0.1070556640625, 0.71533203125, -1.431640625, -0.880859375, 0.338134765625, -0.442626953125, -1.142578125, 0.91943359375, 0.08294677734375, 0.892578125, -0.0289459228515625, 0.75634765625, -0.08843994140625, -1.0712890625, 0.09234619140625, 0.1781005859375, 0.06976318359375, 0.68505859375, -1.107421875, 0.2049560546875, 0.351806640625, 1.564453125, 1.013671875, -0.255126953125, -0.370361328125, -0.11773681640625, -0.3447265625, 0.2978515625, 0.294189453125, 0.468994140625, 1.2880859375, -1.263671875, -0.01139068603515625, -0.360107421875, 1.1328125, 0.955078125, -0.078369140625, -0.6259765625, -0.9677734375, -0.541015625, 1.21484375, 0.180908203125, 0.55908203125, 0.80029296875, -0.97021484375, -0.96728515625, -1.607421875, 0.041595458984375, -0.38525390625, -2.38671875, 0.720703125, 0.82958984375, -0.984375, 0.95703125, 0.37060546875, -0.87890625, -0.10919189453125, 1.5, 0.335205078125, 2.017578125, 1.11328125, -0.67041015625, 1.4755859375, 0.30224609375, -0.7158203125, 0.5244140625, -0.52783203125, 0.47900390625, -1.78515625, -1.0322265625, -1.62109375, 0.81884765625, -1.228515625, 0.338623046875, -0.44921875, -0.9052734375, -0.94091796875, 0.60107421875, 0.76318359375, 0.99853515625, 0.3115234375, 0.1546630859375, 0.74658203125, -0.2734375, 0.572265625, 0.443603515625, 1.0498046875, 1.0390625, -1.7890625, 0.1224365234375, -1.6220703125, -0.7509765625, -0.75390625, 0.2978515625, 0.306884765625, -0.0999755859375, 0.133544921875, -1.041015625, 2.05859375, -0.8291015625, 0.356201171875, -1.115234375, -0.344970703125, -0.96630859375, 2.08203125, -0.2078857421875, 0.939453125, -1.6689453125, -0.736328125, -0.50439453125, -1.142578125, -1.3798828125, 2.189453125, 2.001953125, 0.333251953125, 0.3095703125, -0.7294921875, 0.282470703125, -0.1285400390625, 1.0947265625, -0.490478515625, -0.72998046875, 1.224609375, 1.6201171875, -1.474609375, -0.041656494140625, -1.1796875, 1.7001953125, 1.2392578125, -0.327880859375, 2.404296875, 0.82763671875, 1.75390625, 0.2310791015625, -1.23828125, -0.56640625, 0.480224609375, -1.365234375, 0.023040771484375, -0.26123046875, 1.6708984375, 0.485107421875, 1.4951171875, 0.0478515625, 0.023651123046875, 0.5341796875, -1.265625, -1.0537109375, -1.283203125, -0.89013671875, 0.64794921875, 0.3984375, -0.98779296875, -0.264892578125, 1.5498046875, 0.345703125, 0.468017578125, -0.642578125, -0.0777587890625, -0.6796875, -0.59912109375, 0.333984375, 1.5048828125, -0.73291015625, 1.224609375, 2.03125, 1.6484375, 0.81591796875, -1.5556640625, -2.3046875, 0.72607421875, -0.222412109375, 0.230224609375, -1.888671875, -1.634765625, 0.72802734375, -0.183837890625, -1.728515625, 0.79052734375, -0.7802734375, 0.458251953125, -0.04266357421875, 0.498779296875, -0.65087890625, 1.103515625, 0.587890625, 0.439697265625, 1.529296875, 0.040313720703125, -1.126953125, -1.2880859375, -1.802734375, -1.2333984375, 0.01477813720703125, -0.71875, -1.685546875, 0.8955078125, -0.265625, -1.3193359375, -1.017578125, 0.53369140625, 0.08538818359375, 1.7001953125, 0.7060546875, -2.302734375, 2.224609375, 1.8076171875, -0.17041015625, -0.1231689453125, -0.701171875, 0.83740234375, 0.313720703125, -0.8125, 1.2783203125, 0.6904296875, 0.385498046875, 0.1708984375, 1.2666015625, 0.202880859375, 0.94873046875, -0.1142578125, -0.17236328125, -1.29296875, 0.69287109375, -0.304443359375, 0.2120361328125, 0.06219482421875, -1.0654296875, -0.548828125, 0.65087890625, 0.025360107421875, 0.9091796875, -0.437255859375, 1.1982421875, -0.99853515625, -0.023406982421875, 1.0537109375, -1.349609375, -0.98193359375, -0.419677734375, 0.564453125, -0.271728515625, -1.5126953125, 0.326171875, 0.52099609375, -1.5087890625, -0.82666015625, 0.51708984375, 0.8779296875, -0.66552734375, -0.0887451171875, -0.9814453125, 0.53173828125, 1.0, 0.61572265625, -0.97802734375, -0.6982421875, 0.09564208984375, -0.52978515625, 0.85302734375, -0.6455078125, -1.7431640625, 0.375, 1.9287109375, -0.9755859375, -1.658203125, -0.8505859375, -1.44921875, 1.619140625, -1.6435546875, 0.71240234375, 0.53173828125, 0.671875, 0.51708984375, 0.7412109375, 1.3359375, -0.6728515625, -0.9970703125, -1.4228515625, -0.2188720703125, -1.2373046875, 0.43017578125, -0.469482421875, 0.1356201171875, -1.5927734375, 0.39501953125, -0.21142578125, -0.463134765625, 0.2222900390625, 0.7783203125, -0.1611328125, -0.39453125, -0.308349609375, 0.198486328125, 0.62109375, 0.6201171875, 0.6279296875, -0.1728515625, 1.0634765625, 0.46923828125, 0.451416015625, -0.06378173828125, 0.74853515625, -2.826171875, 0.368896484375, -0.6416015625, -0.225830078125, 0.796875, 0.37060546875, 0.415771484375, -1.2177734375, -0.18896484375, 0.99658203125, 0.384765625, -0.775390625, -0.3740234375, 1.6982421875, -1.078125, 0.896484375, -1.853515625, -0.423828125, 0.397705078125, -0.1142578125, 1.033203125, 0.05291748046875, 0.036041259765625, 0.53515625, -0.23486328125, -0.7705078125, 0.058990478515625, -0.5654296875, 1.1708984375, -1.240234375, -1.05859375, 0.55029296875, 0.2471923828125, -1.8740234375, -0.306640625, -1.8994140625, 0.71337890625, -0.313720703125, -2.0, -0.27197265625, 1.2021484375, -0.072998046875, -2.087890625, 0.0887451171875, 0.80859375, -0.61376953125, -0.990234375, 0.381103515625, -1.779296875, 0.292724609375, 0.427001953125, 0.46044921875, 0.337890625, -0.5302734375, 0.07928466796875, -1.216796875, 0.053466796875, -0.66357421875, -0.73095703125, 2.556640625, 0.1494140625, -0.9482421875, 0.7470703125, 0.88330078125, 0.7080078125, -0.3408203125, 0.0284271240234375, 2.265625, -0.85888671875, 0.2244873046875, -1.2978515625, 0.052215576171875, -0.06402587890625, 1.1796875, -0.6806640625, 1.03125, 0.332275390625, -0.10223388671875, 0.90185546875, -0.2362060546875, 1.3076171875, 0.9404296875, 0.6123046875, -2.51171875, -1.0791015625, -1.369140625, 0.65234375, 0.2437744140625, 1.248046875, 2.236328125, 0.040313720703125, -0.1336669921875, 0.416015625, 1.458984375, 0.71142578125, 0.8984375, 0.149658203125, 0.525390625, -1.05859375, 0.919921875, -0.77783203125, -1.919921875, 1.1123046875, -0.66015625, 1.1708984375, 0.958984375, -0.039886474609375, -0.912109375, 0.260009765625, -1.21484375, 0.95751953125, 2.0546875, 0.313720703125, -0.9921875, 0.78173828125, -1.4990234375, -1.583984375, 1.4482421875, 0.497314453125, 0.8837890625, -0.390625, -0.52587890625, 0.03924560546875, -1.0615234375, 0.71044921875, -2.43359375, -0.5283203125, -0.457763671875, 0.12152099609375, 0.29150390625, 0.7978515625, 0.86279296875, -1.0791015625, 0.303955078125, -1.240234375, -0.41943359375, 1.166015625, 0.70703125, -1.2392578125, 0.94873046875, 0.61669921875, 1.0107421875, -1.078125, 0.243408203125, 0.461669921875, 0.306640625, 0.312744140625, -0.7587890625, -1.0224609375, -0.08544921875, 0.451416015625, 2.03125, -0.7421875, -0.89013671875, -0.83740234375, -0.279052734375, -0.9267578125, -0.82080078125, 0.286376953125, 1.0341796875, -0.276123046875, -0.048553466796875, 1.072265625, -0.1063232421875, -0.234130859375, -0.1278076171875, -0.85693359375, -0.298828125, 0.71044921875, -0.79345703125, 0.29248046875, -2.00390625, -2.033203125, 0.466552734375, -0.3271484375, 0.029541015625, -0.1563720703125, 0.340087890625, -0.435546875, -0.3916015625, -0.39697265625, 0.83837890625, -0.935546875, -0.982421875, 1.5185546875, 1.9443359375, 0.212158203125, -0.332275390625, -1.6806640625, 0.67578125, -0.1854248046875, 0.2325439453125, -0.47119140625, 0.33154296875, 0.69677734375, -1.7177734375, -1.271484375, 0.37255859375, 0.85009765625, -0.1181640625, -0.951171875, 2.00390625, -0.55517578125, 0.06890869140625, -0.80517578125, -1.146484375, -0.195556640625, 1.36328125, 1.5693359375, -0.90478515625, 1.13671875, 0.3759765625, 0.9833984375, 1.599609375, -0.346923828125, -0.6083984375, 0.112548828125, -0.42529296875, -0.4130859375, -1.029296875, -0.52880859375, 1.8310546875, 0.294677734375, -0.406982421875, 2.556640625, 0.76220703125, -0.439453125, -1.09765625, 0.59033203125, 1.240234375, 0.72900390625, 0.56689453125, 0.130126953125, -0.03948974609375, -2.587890625, -0.89111328125, -1.4794921875, 0.142578125, 1.7822265625, -2.205078125, -1.037109375, -2.109375, -1.2626953125, -0.95166015625, -0.1025390625, -0.492919921875, -0.404541015625, 1.0048828125, 0.70947265625, 0.0885009765625, 1.166015625, -1.33984375, -1.5947265625, -2.365234375, 1.48046875, -0.06585693359375, -0.441162109375, 1.263671875, 0.61767578125, 0.67919921875, -1.4111328125, -0.81005859375, 0.273193359375, 0.86572265625, -0.56494140625, -1.0498046875, 0.1563720703125, -0.5078125, -0.458984375, -0.3681640625, 1.1845703125, 1.0029296875, -0.030426025390625, 0.2442626953125, 1.3408203125, -1.0947265625, -0.5673828125, 0.541015625, 1.0146484375, -0.69140625, -0.1861572265625, -2.22265625, 0.0201416015625, 0.044708251953125, 0.77490234375, -2.12890625, 1.93359375, -0.46142578125, 0.53125, 0.440185546875, -0.62890625, -0.35986328125, -0.07421875, -1.3193359375, 1.3505859375, 0.387939453125, -0.0411376953125, 1.0078125, -1.0830078125, -2.193359375, -0.7255859375, 0.63330078125, -2.8984375, 0.282958984375, -0.505859375, 0.59912109375, 0.8046875, 0.5126953125, 0.77783203125, -0.58935546875, 0.384521484375, -0.70361328125, 2.681640625, 1.212890625, 1.474609375, 1.1806640625, -1.5234375, -0.3046875, 0.9873046875, 0.26171875, 0.130126953125, -0.30908203125, 0.6181640625, -0.5673828125, -0.04345703125, 0.449462890625, 2.11328125, 1.3427734375, 0.25927734375, 0.1158447265625, -0.1378173828125, 1.0849609375, 2.712890625, -0.5791015625, 0.341796875, 0.8291015625, 0.380859375, -0.32421875, -1.080078125, -0.46044921875, 0.0557861328125, 0.833984375, -0.371826171875, 0.9912109375, -0.482666015625, 1.2373046875, 1.109375, -0.60595703125, -1.2880859375, -0.5146484375, 0.9951171875, 0.270263671875, -0.8486328125, 1.0048828125, -0.83349609375, -0.2325439453125, 0.72314453125, -0.1480712890625, 0.91357421875, 1.0048828125, 0.364501953125, -2.03125, -0.032806396484375, -0.30419921875, -0.313232421875, 0.8505859375, -0.56884765625, 0.1002197265625, -0.275146484375, -1.38671875, -0.0772705078125, 1.552734375, -1.5380859375, -1.228515625, -0.6923828125, -0.472412109375, -0.740234375, 1.81640625, -0.37451171875, -0.64453125, -0.634765625, 0.11700439453125, -0.1431884765625, 1.3486328125, -0.8486328125, -0.74951171875, -2.359375, -1.3271484375, -1.1025390625, -0.252197265625, -0.33203125, -1.0087890625, -0.2303466796875, 0.1976318359375, 1.0166015625, 0.5546875, -1.5673828125, 1.0791015625, 0.253173828125, 0.39697265625, -0.70703125, -0.2427978515625, -0.796875, -0.3896484375, -1.0703125, 1.4755859375, -0.1611328125, -0.051910400390625, -1.646484375, -0.98193359375, -1.0126953125, 0.876953125, 1.4609375, 2.001953125, -1.7685546875, -0.1917724609375, 0.1197509765625, -0.033905029296875, 3.037109375, 0.407470703125, -0.529296875, -1.0537109375, -0.285400390625, 0.031829833984375, 1.0107421875, -0.09539794921875, 2.216796875, 0.546875, -0.60693359375, 0.1728515625, -0.4765625, 1.2666015625, 0.178955078125, -0.5712890625, -0.53466796875, 0.62646484375, 0.088623046875, 1.646484375, 0.4072265625, -0.888671875, 1.1826171875, 0.412109375, 1.5087890625, 0.2181396484375, -0.91015625, -0.2587890625, 0.00225067138671875, 1.2861328125, -0.1192626953125, 0.6826171875, 1.3779296875, -1.451171875, -1.724609375, 0.87744140625, -1.3037109375, -1.7890625, -0.650390625, 0.26513671875, 0.274169921875, -0.5126953125, -1.158203125, 0.07403564453125, 0.04345703125, 0.6142578125, -0.841796875, -1.7607421875, -1.07421875, 0.0284881591796875, 2.1953125, 0.0019702911376953125, -0.61474609375, 1.478515625, -0.5859375, 0.08123779296875, -0.36767578125, 0.30029296875, 1.57421875, 0.25048828125, -1.298828125, -2.6328125, -0.1776123046875, 0.199951171875, 0.285400390625, -1.0380859375, 0.72802734375, 0.0255889892578125, -1.00390625, 0.62451171875, -0.60595703125, 0.67822265625, -2.31640625, -0.236572265625, -1.072265625, 3.892578125, 1.9326171875, 0.471435546875, 0.2359619140625, -0.53271484375, -0.42138671875, -1.8544921875, -0.29541015625, 0.332275390625, 0.0899658203125, -0.08258056640625, -1.5458984375, 1.580078125, -1.0966796875, 0.288818359375, -1.2822265625, 0.380126953125, 0.1357421875, 0.90673828125, 0.90478515625, -1.580078125, 0.13330078125, -0.344482421875, 0.40966796875, 0.5673828125, 2.080078125, -0.93701171875, -0.389404296875, 0.2474365234375, -0.08343505859375, -0.5458984375, -0.376220703125, 0.92724609375, -1.17578125, 1.20703125, -2.306640625, 0.391357421875, -1.95703125, 0.330810546875, -0.6142578125, -0.455322265625, -0.333251953125, 0.78515625, -0.59130859375, -0.293212890625, 0.60888671875, 1.8056640625, 0.236083984375, -0.310302734375, 0.09344482421875, -1.4013671875, -0.484375, -1.5703125, 0.59521484375, 0.315673828125, -0.9130859375, -0.63525390625, 0.255126953125, -1.4453125, 1.13671875, -1.330078125, -1.265625, 1.4462890625, 1.0673828125, 0.48486328125, -2.0546875, -0.375, 0.35498046875, -0.41357421875, 2.1796875, -0.285888671875, -0.443115234375, 1.044921875, 0.681640625, 0.71875, -0.157470703125, 0.0202178955078125, 0.27392578125, -0.433837890625, 0.0787353515625, -1.0986328125, 0.58740234375, -0.68017578125, -0.69580078125, -0.84228515625, 0.78564453125, -1.7294921875, 0.9365234375, 0.56201171875, 0.9755859375, -1.6162109375, 1.076171875, -0.36328125, 0.2342529296875, -1.681640625, -0.69921875, 0.671875, -0.4150390625, -0.75244140625, 1.4365234375, -1.0634765625, 0.358642578125, 0.68212890625, 0.5986328125, -0.495849609375, -0.5439453125, -0.01151275634765625, 1.0712890625, -0.509765625, 0.37548828125, -0.4736328125, -0.55029296875, 0.86474609375, 0.2568359375, -0.54541015625, 0.28662109375, -0.802734375, -0.9208984375, -0.66357421875, 0.203857421875, 1.0048828125, 0.42919921875, -0.9462890625, 0.98974609375, 0.53466796875, -1.189453125, -0.2354736328125, -0.2880859375, 1.1904296875, -1.2734375, -0.95166015625, 0.00525665283203125, -1.6962890625, 1.9482421875, -1.578125, 1.2998046875, -1.3017578125, -0.5625, 1.5107421875, 1.24609375, 1.48828125, 0.38427734375, 0.280517578125, -1.1083984375, 0.358642578125, -0.2978515625, -0.09527587890625, -0.82177734375, 1.255859375, -0.7646484375, 0.68603515625, -0.098876953125, -0.83447265625, 1.6611328125, -1.869140625, -1.1728515625, 0.80078125, 1.943359375, -2.291015625, 0.58544921875, -0.0155792236328125, -0.1759033203125, 0.413818359375, 1.1826171875, 1.3271484375, 1.5830078125, 0.7431640625, -0.73681640625, 0.418701171875, 0.45361328125, 1.5478515625, -1.234375, -1.08203125, -1.1494140625, -0.2344970703125, -0.2161865234375, 0.62109375, 0.986328125, -1.3154296875, -0.438232421875, 0.544921875, -0.072998046875, 0.9306640625, -1.4267578125, 0.6787109375, -2.509765625, 0.86083984375, -0.280029296875, 0.9267578125, -0.7724609375, 0.650390625, -1.1083984375, -1.015625, -0.548828125, -0.8076171875, -1.0791015625, -0.286865234375, -1.208984375, -0.036773681640625, 0.94921875, -0.73388671875, -0.0145721435546875, -1.30078125, -1.697265625, -0.36474609375, 1.2275390625, -0.28076171875, 0.740234375, -0.0621337890625, -0.002780914306640625, -0.94580078125, -1.2177734375, 1.0361328125, 0.72265625, -0.217529296875, 0.47607421875, 0.407470703125, -0.91259765625, -0.343017578125, -2.5078125, 1.23046875, -1.435546875, 0.03790283203125, -1.298828125, 0.481689453125, 1.197265625, 0.41552734375, 1.240234375, -0.626953125, -0.3359375, 1.0859375, 0.2095947265625, -0.68359375, -0.509765625, -0.1512451171875, -0.77685546875, -1.138671875, 0.5576171875, -0.10076904296875, 0.69091796875, 1.06640625, 2.087890625, -1.4365234375, 0.9697265625, -1.30078125, -1.2880859375, -0.5205078125, 1.443359375, 1.126953125, -0.81640625, 0.440673828125, 1.7431640625, -0.681640625, -1.0546875, -0.67529296875, 0.5947265625, 0.74658203125, 0.3759765625, -0.3740234375, -0.509765625, 0.6328125, -0.1583251953125, -0.10400390625, -0.97021484375, 0.98828125, 0.892578125, -0.27783203125, -1.5458984375, -0.50390625, 0.77734375, -0.8974609375, -1.3876953125, -0.0941162109375, 0.17822265625, -1.2109375, -2.677734375, -0.9033203125, 1.130859375, 0.002872467041015625, -1.625, 1.0556640625, 0.1715087890625, -0.1763916015625, -0.3408203125, -0.50244140625, -0.60107421875, -0.1456298828125, -0.09686279296875, 0.7421875, -1.3857421875, -0.045318603515625, -0.41943359375, -0.31494140625, 0.365478515625, 0.7109375, 0.7158203125, -0.23876953125, -0.90478515625, 1.2509765625, 0.05999755859375, -0.98876953125, -0.384521484375, -0.11419677734375, 0.60791015625, 0.6298828125, 0.96142578125, -0.7060546875, 0.7763671875, -0.9072265625, -0.32470703125, -1.3388671875, 0.5029296875, 0.90185546875, 0.1400146484375, -1.787109375, -0.8203125, -0.1053466796875, 0.055877685546875, 0.5419921875, -0.56396484375, 1.4443359375, 0.73681640625, -0.376953125, 0.1280517578125, 2.61328125, -2.654296875, -0.21728515625, -0.454345703125, 1.5029296875, 0.03204345703125, -0.7998046875, -1.904296875, -0.88720703125, 2.17578125, -0.4794921875, -1.7041015625, 0.451416015625, 0.95751953125, -0.1439208984375, -0.28173828125, -0.77880859375, 1.265625, 1.5322265625, 1.22265625, -0.7763671875, 1.23046875, 1.501953125, -0.044189453125, -1.99609375, 1.20703125, -0.85498046875, 0.60107421875, -1.0947265625, -0.77783203125, 0.5439453125, 0.5224609375, -0.5537109375, 2.375, -1.564453125, -1.1806640625, 0.58447265625, 2.375, 0.48193359375, 0.94775390625, -0.033233642578125, -0.52587890625, -0.274658203125, 0.455810546875, -0.67529296875, 0.432373046875, -0.525390625, -0.56396484375, -0.42431640625, -1.2763671875, 1.3310546875, 1.0146484375, 1.271484375, 0.382568359375, -0.63232421875, 0.50634765625, -0.64013671875, 2.865234375, 0.11651611328125, 0.52685546875, 0.1494140625, -0.077880859375, -1.0068359375, -0.6171875, -1.3017578125, 0.560546875, -0.68994140625, -0.046417236328125, -0.253662109375, 0.80615234375, -0.669921875, 1.1279296875, 1.072265625, 1.677734375, -1.1240234375, 1.255859375, 0.357177734375, -0.88134765625, 1.9814453125, 0.876953125, -0.70068359375, 0.2275390625, 0.490966796875, -0.170654296875, -1.8974609375, 1.140625, -0.44140625, -1.0712890625, 0.36376953125, -1.130859375, -0.059478759765625, -0.2388916015625, -0.83740234375, -0.88818359375, 1.306640625, 0.040435791015625, -0.2646484375, 0.0858154296875, -0.51171875, -0.1959228515625, -0.50634765625, 0.401611328125, 0.77392578125, -2.537109375, -0.23681640625, -1.341796875, 2.66015625, 0.026123046875, 1.9658203125, -0.450927734375, -1.3681640625, 1.3681640625, 1.6025390625, 0.1727294921875, -0.64404296875, -1.7822265625, -0.4541015625, -0.666015625, 0.744140625, 0.89404296875, -0.80810546875, -2.345703125, -0.8173828125, 1.337890625, -1.4521484375, -0.7734375, -0.1795654296875, -2.16796875, 0.55859375, 1.1826171875, 0.79052734375, 0.43603515625, -0.79638671875, 1.732421875, -0.1265869140625, -1.44921875, -0.55029296875, -1.734375, -0.0137939453125, 0.62890625, -0.6484375, -0.141357421875, 1.1201171875, -0.269287109375, 2.69140625, -0.49365234375, -0.8271484375, -0.435302734375, 0.0821533203125, 1.205078125, -0.98828125, 1.333984375, 0.84521484375, 0.390625, 0.415771484375, 0.25927734375, -0.1348876953125, -0.318359375, -0.5673828125, -1.17578125, 0.53076171875, 0.70166015625, 1.5439453125, 0.210205078125, -1.5087890625, -0.381591796875, -0.09210205078125, -0.6787109375, -0.51171875, 0.058013916015625, -0.86865234375, -1.958984375, 1.0849609375, -0.187255859375, 0.56103515625, -0.88330078125, -0.861328125, 1.1279296875, 1.1240234375, 0.1087646484375, -0.0036334991455078125, -0.626953125, 0.385009765625, -0.1417236328125, -1.7109375, 1.6630859375, -0.52490234375, 1.2724609375, -0.79638671875, 2.3203125, -1.3857421875, -0.60791015625, -0.150390625, -0.1905517578125, -0.32177734375, -1.025390625, 0.5908203125, -1.744140625, 1.8056640625, -1.5048828125, -0.7373046875, 0.1507568359375, -0.3583984375, -1.072265625, -0.65087890625, 0.1839599609375, -0.498291015625, -0.87109375, -1.537109375, -0.40673828125, -0.50537109375, -0.65380859375, 0.87451171875, 0.24365234375, -1.326171875, 0.1962890625, -0.2340087890625, -0.59765625, 0.1593017578125, -1.484375, 1.7119140625, -0.85986328125, -0.8134765625, 0.7060546875, 0.50537109375, 2.283203125, 0.8359375, -1.7470703125, 0.06353759765625, -0.59716796875, -0.52587890625, 0.0498046875, -0.8525390625, 0.56884765625, -0.7138671875, 1.53515625, -1.4189453125, -0.248779296875, -0.182373046875, -0.084228515625, -0.341796875, 1.7666015625, 0.783203125, 0.0928955078125, 0.43212890625, -0.798828125, -0.1883544921875, 0.95703125, -0.6357421875, 0.6201171875, 0.57568359375, -0.68115234375, -0.416259765625, -2.39453125, -1.0078125, -1.7451171875, -2.583984375, -0.429931640625, 0.62646484375, -0.12548828125, 2.271484375, -0.0667724609375, 0.47802734375, 0.71142578125, 1.0185546875, 0.07159423828125, 0.97412109375, 1.09375, -0.53076171875, 0.84033203125, 0.58544921875, 1.021484375, 1.251953125, 0.6552734375, 1.033203125, -0.48779296875, 0.41552734375, -0.59716796875, -0.159912109375, -0.75244140625, 0.2958984375, -1.576171875, -0.1768798828125, -0.87158203125, 1.384765625, -0.60205078125, 0.52783203125, 0.68994140625, -1.0234375, -0.62451171875, -0.79931640625, -0.560546875, -0.27001953125, -1.1435546875, 1.5205078125, 0.460693359375, 0.29736328125, -0.6982421875, 1.328125, 1.1630859375, 1.2958984375, 1.095703125, -1.20703125, 0.002910614013671875, -0.359130859375, -0.258544921875, 1.25, -0.8310546875, -0.8671875, 0.54150390625, 0.425537109375, -0.109375, -0.1588134765625, -0.12548828125, 0.2470703125, -0.04327392578125, -0.82275390625, 0.90283203125, 0.389404296875, 1.03125, -1.716796875, 0.65869140625, 0.7158203125, -0.68359375, -0.39794921875, 0.1331787109375, 1.6171875, -0.299560546875, 0.140625, -0.335693359375, 0.0894775390625, 0.11090087890625, -0.30712890625, -1.083984375, 0.95068359375, 0.46875, -2.24609375, -0.178466796875, 0.470947265625, -1.4482421875, -0.37353515625, 0.1680908203125, 0.6416015625, 1.2265625, 0.61083984375, 1.0087890625, -0.052734375, 0.728515625, 1.7822265625, 1.064453125, -0.673828125, -0.6796875, 0.474853515625, 0.08245849609375, -0.1854248046875, 0.54052734375, -0.082275390625, 0.255126953125, 1.0302734375, 0.0226898193359375, -0.27490234375, -1.2412109375, -3.177734375, -1.1572265625, 0.058563232421875, -0.3349609375, -0.43798828125, 0.1331787109375, 0.8603515625, 0.45703125, 2.326171875, -0.63623046875, 0.382080078125, 0.16162109375, 0.30908203125, 1.5673828125, 0.2861328125, -0.345703125, -0.1883544921875, -0.84716796875, -0.62841796875, 1.6435546875, -0.80615234375, -2.10546875, 0.7138671875, -0.900390625, -0.421630859375, -1.1611328125, -0.76171875, -1.759765625, 0.970703125, -1.435546875, 0.88037109375, -0.29052734375, -1.8564453125, -0.8046875, 0.29443359375, -3.2109375, -0.92724609375, 0.348876953125, 1.1962890625, -0.306884765625, -0.71923828125, -0.4453125, -1.169921875, 0.552734375, 0.85400390625, 0.146240234375, 1.736328125, -0.72802734375, 0.45068359375, -1.2470703125, -0.69287109375, 0.55712890625, -0.120849609375, -0.01059722900390625, -1.3623046875, 0.271484375, -0.935546875, 0.75048828125, 0.338623046875, -0.84912109375, 0.81787109375, -1.6142578125, -0.1632080078125, -0.1585693359375, -0.11480712890625, -0.70849609375, 1.046875, 0.1339111328125, 0.63623046875, -1.5078125, 0.352294921875, -0.479248046875, 2.482421875, -0.302978515625, -0.85400390625, 0.23388671875, -1.17578125, 0.595703125, 0.4921875, -0.11480712890625, -0.486328125, 0.2734375, -1.8525390625, -1.3583984375, 0.828125, 1.3798828125, -0.220947265625, 2.212890625, -0.47265625, -2.458984375, 2.970703125, -0.537109375, 1.5244140625, 1.46484375, -1.23828125, 0.1845703125, -0.309326171875, -1.3583984375, -1.689453125, -0.268310546875, -0.497314453125, -1.078125, 1.3662109375, -0.220947265625, -1.88671875, 0.720703125, 0.48583984375, -0.437255859375, -0.294677734375, 0.3291015625, -1.6201171875, -0.168701171875, 1.8515625, 0.358154296875, 1.15234375, -0.619140625, 1.6826171875, -1.3779296875, 1.0234375, 0.271240234375, -0.87255859375, -1.033203125, -0.1282958984375, 0.6279296875, -0.135986328125, 0.311767578125, 0.1495361328125, 0.669921875, 1.6015625, -0.28125, -0.2001953125, 0.62451171875, 0.931640625, 0.6865234375, 0.39111328125, -0.36669921875, 1.564453125, -0.91796875, 0.787109375, -0.427734375, 0.7958984375, 1.525390625, -0.97509765625, -0.53369140625, -0.4873046875, 0.765625, -0.36279296875, -0.40966796875, -0.15185546875, 1.3974609375, 0.75048828125, -0.478759765625, 0.10723876953125, -1.1240234375, 0.04461669921875, -1.109375, 0.0029239654541015625, -0.3046875, 1.828125, -3.052734375, 0.7724609375, -0.46728515625, -1.5810546875, 0.9892578125, 0.436279296875, 0.09918212890625, -1.3359375, -1.28125, 0.97998046875, -1.166015625, -1.9326171875, 0.79150390625, 0.8857421875, -2.052734375, 0.89013671875, 0.1368408203125, 0.07928466796875, 0.254638671875, -0.8837890625, -1.2265625, -1.6904296875, 0.0261383056640625, -0.23388671875, 0.45068359375, 0.0222320556640625, 0.6484375, -1.326171875, -1.21875, 0.2015380859375, -0.11224365234375, -1.3095703125, -0.478515625, -0.286865234375, -0.1287841796875, 0.4326171875, 1.7822265625, -2.2578125, -1.013671875, 2.583984375, -0.65576171875, -0.493896484375, -0.495849609375, 1.0068359375, -0.87646484375, 0.7333984375, 0.0107879638671875, -0.71337890625, 0.763671875, 1.947265625, 1.17578125, -0.96630859375, -0.1053466796875, 0.369140625, 0.5849609375, 2.583984375, -0.5009765625, 0.65087890625, -0.859375, -0.266845703125, -0.63427734375, -1.03515625, 0.72998046875, -1.130859375, -0.78271484375, 1.2236328125, -2.046875, 0.7275390625, 0.1326904296875, -2.48828125, -0.5400390625, -0.045166015625, -0.05902099609375, -0.541015625, -1.107421875, -0.25341796875, -0.1744384765625, 1.0810546875, -1.56640625, 0.70947265625, 0.7919921875, 0.3837890625, 0.231201171875, 1.044921875, -1.3515625, 0.4833984375, -0.783203125, -1.7265625, -2.068359375, 0.259521484375, -0.60888671875, 1.4140625, 0.84765625, -0.29052734375, -0.72607421875, 0.8232421875, -2.091796875, -1.052734375, 0.22021484375, 0.3359375, 1.482421875, -1.244140625, 1.0546875, -1.978515625, 0.4658203125, 2.634765625, -0.28564453125, 1.2685546875, 0.231689453125, 0.368408203125, 0.70556640625, 0.6474609375, 1.080078125, 1.0498046875, 0.7119140625, 1.923828125, -1.8076171875, 1.15234375, -0.630859375, -0.4736328125, 0.492919921875, -0.2086181640625, 0.7490234375, -0.67578125, -0.359130859375, -0.9716796875, 2.11328125, -0.62451171875, -0.78662109375, 0.90380859375, -0.59765625, 1.390625, -0.64453125, -1.0390625, -0.94677734375, -0.298828125, -0.5517578125, 0.33203125, -0.2386474609375, -1.3525390625, -0.09228515625, -0.7470703125, 1.6142578125, 0.0238189697265625, -0.2315673828125, 0.45263671875, -0.775390625, -0.1375732421875, 1.17578125, -0.326904296875, -0.84814453125, -0.5458984375, -1.0419921875, -0.473876953125, -0.427978515625, -0.5458984375, -0.74169921875, 0.468505859375, 0.1822509765625, 0.292236328125, 1.2646484375, 0.09906005859375, -0.96337890625, 2.427734375, -1.3837890625, -1.21875, -0.78466796875, 1.3916015625, -1.3984375, 0.2041015625, -0.115966796875, 1.8330078125, -2.361328125, 0.89990234375, -0.026947021484375, -1.3642578125, 1.0712890625, 1.5263671875, 1.2890625, -0.98974609375, 0.853515625, -0.435546875, -0.5712890625, 0.9462890625, 2.630859375, 0.82421875, 1.177734375, -0.88818359375, -0.71533203125, 0.82470703125, 0.1168212890625, 1.818359375, -1.0009765625, 0.5126953125, 1.4951171875, 0.54443359375, -0.517578125, 1.8916015625, 0.271240234375, 1.5546875, -0.053314208984375, 0.56884765625, -0.117919921875, -0.98779296875, 0.479248046875, 0.396728515625, 1.1025390625, 0.73828125, -0.1839599609375, -1.1884765625, -0.283447265625, 0.0199432373046875, -1.3486328125, 1.2333984375, 0.93798828125, -1.5029296875, 2.197265625, -0.04290771484375, 0.4580078125, 1.7998046875, 0.09759521484375, 0.798828125, 1.2626953125, -0.2196044921875, 1.306640625, 0.927734375, 1.296875, 0.599609375, -0.71142578125, -1.1005859375, -0.2152099609375, -0.9287109375, -0.1181640625, 0.18115234375, 2.923828125, 1.4345703125, -0.85693359375, 0.0110626220703125, -0.62255859375, 2.3515625, 0.314453125, 0.242919921875, 0.142333984375, 0.0111846923828125, -0.004734039306640625, -0.52197265625, 0.80712890625, 1.0263671875, 0.04541015625, 1.40625, -1.2724609375, 0.287841796875, -0.265380859375, 0.79150390625, 1.2841796875, 0.2086181640625, -0.68310546875, 0.368896484375, 0.1751708984375, 0.52685546875, 0.144287109375, 0.21484375, -0.7548828125, -0.024200439453125, -0.2103271484375, -1.4951171875, -0.389404296875, -0.98486328125, 0.304443359375, 1.0908203125, -0.501953125, 0.309326171875, 1.5849609375, -0.0841064453125, -0.1116943359375, 1.1484375, 0.5107421875, 0.669921875, -0.313232421875, -0.10858154296875, 0.7783203125, 0.46435546875, -0.484375, -0.280029296875, 0.172119140625, 1.587890625, 1.0234375, 0.3955078125, 0.2325439453125, 0.5390625, -2.251953125, -0.55712890625, 0.2900390625, -0.5224609375, 1.6318359375, 1.2685546875, -0.943359375, -0.9599609375, -0.1051025390625, 0.728515625, 0.63330078125, 0.72216796875, 1.5078125, 0.12274169921875, 0.344970703125, -0.48486328125, -0.382568359375, -0.35546875, 1.03125, 0.1441650390625, 1.4775390625, 0.2078857421875, -0.438232421875, 1.7275390625, 1.6875, 2.419921875, 1.24609375, 0.0565185546875, 0.85693359375, 1.6865234375, -0.1693115234375, -0.64892578125, 0.270751953125, -1.1962890625, -0.3076171875, -0.320556640625, -0.126220703125, 0.273193359375, 0.84375, 1.99609375, 0.7783203125, 2.05859375, -0.568359375, -0.53173828125, -0.5537109375, 0.059295654296875, 1.01171875, -0.373046875, 0.109619140625, -1.5888671875, 0.77880859375, -0.10003662109375, -0.71484375, 0.177490234375, -0.412353515625, -1.287109375, 0.297119140625, 1.7021484375, -1.5947265625, 1.2314453125, -0.178955078125, -1.1875, -1.865234375, 0.236083984375, -1.0654296875, -0.541015625, -3.2265625, -0.935546875, -0.80126953125, 1.8935546875, 1.201171875, 1.0263671875, 0.70947265625, 0.73828125, -1.3740234375, 0.07281494140625, 1.9716796875, -0.1385498046875, -0.48681640625, 0.185302734375, -0.50048828125, -1.5146484375, 0.003170013427734375, 1.623046875, -2.4296875, 0.50341796875, -1.3740234375, 0.86279296875, 0.2049560546875, 0.455810546875, 0.86279296875, 0.06817626953125, -0.49755859375, 0.0219879150390625, 0.380126953125, 0.99951171875, 1.1484375, 1.6513671875, 1.41796875, -0.466064453125, 1.3349609375, -0.411376953125, -0.334716796875, -0.0665283203125, 1.2060546875, -0.46337890625, -0.912109375, 0.328125, -0.75830078125, 1.501953125, -0.7314453125, 1.3486328125, 0.088623046875, -1.169921875, 0.76025390625, -0.209228515625, -0.6904296875, 1.48046875, 0.193115234375, 1.0087890625, 0.2200927734375, 0.45361328125, 0.943359375, 0.297607421875, -0.455810546875, -0.9560546875, -1.224609375, -0.402587890625, 0.91943359375, 2.232421875, -0.60986328125, -0.53271484375, 1.0771484375, -0.5634765625, 0.54931640625, -0.6474609375, 0.49755859375, -1.1513671875, -1.294921875, 1.578125, -1.314453125, 0.7509765625, -0.47607421875, 0.6416015625, 1.4326171875, 1.96484375, 0.1856689453125, -0.3232421875, -0.223388671875, -0.7978515625, 0.9619140625, -0.501953125, 0.55224609375, -0.8271484375, 0.51171875, -0.354248046875, 0.7236328125, 1.0537109375, 0.513671875, -0.32421875, 0.32080078125, 0.97412109375, 1.755859375, -1.267578125, -0.274169921875, -0.46875, 1.3701171875, -0.165283203125, -0.7685546875, -0.51171875, 0.291015625, 0.92724609375, -0.2388916015625, -0.71533203125, 1.2666015625, 0.44775390625, -0.71923828125, 0.308349609375, 0.0133819580078125, 1.244140625, 0.0103302001953125, -0.40478515625, 0.13818359375, 0.8701171875, -1.1328125, -1.021484375, 2.330078125, 0.452392578125, -0.448486328125, 0.79052734375, 0.366455078125, 0.61181640625, 2.046875, -0.70068359375, 0.0980224609375, -1.1962890625, 1.0107421875, -0.10736083984375, 0.16796875, 1.6064453125, 0.65234375, -0.61767578125, -0.4619140625, -0.59814453125, -0.206787109375, 1.1171875, -0.295654296875, -0.79541015625, 0.9228515625, -1.248046875, 0.68701171875, -0.0904541015625, -0.1397705078125, -0.430908203125, 0.349609375, -0.35302734375, 0.69873046875, -0.0235443115234375, 0.56298828125, -0.5341796875, 0.298828125, 0.3056640625, 0.57080078125, 0.50830078125, -0.49169921875, 1.3779296875, 0.50537109375, 0.2734375, -0.791015625, -1.501953125, -0.45458984375, 0.53076171875, -0.44482421875, -2.087890625, 1.8359375, 0.1732177734375, 0.1514892578125, 0.3359375, 1.005859375, 0.734375, -1.884765625, -0.0355224609375, -0.66748046875, 1.083984375, 1.26953125, -0.85107421875, -1.154296875, -0.022674560546875, -2.0859375, 0.213623046875, -0.2308349609375, 0.8095703125, 0.1187744140625, 1.2978515625, 0.77783203125, 0.407958984375, -0.5927734375, -0.53955078125, -0.398681640625, 0.89990234375, 0.5, -0.94775390625, 0.259765625, -0.7216796875, -0.284423828125, 0.33056640625, 1.2236328125, 0.055633544921875, 0.17236328125, 0.77294921875, 0.08392333984375, -0.6748046875, -1.513671875, -0.4912109375, -1.890625, -0.65185546875, 1.458984375, -0.484375, 0.63037109375, -0.16455078125, 0.650390625, -1.001953125, 1.6630859375, -1.9853515625, 2.73046875, -0.98388671875, -1.060546875, -0.77001953125, 0.368896484375, 0.368896484375, 0.79248046875, 0.04986572265625, 1.5908203125, -0.055877685546875, -0.33837890625, -0.43798828125, 0.361083984375, 1.41015625, 0.06744384765625, -1.404296875, -0.9423828125, 0.77099609375, 0.7529296875, 0.25634765625, 1.6708984375, 0.44482421875, -0.533203125, 1.2080078125, 0.491943359375, 0.1898193359375, -0.708984375, -1.0087890625, 0.84423828125, 0.1358642578125, 0.57373046875, 0.94970703125, -1.130859375, 0.84912109375, -0.265625, 0.226318359375, -1.4150390625, -0.71240234375, 0.5654296875, -0.25, 1.1728515625, -0.2249755859375, -1.1142578125, -0.53173828125, -0.1444091796875, -0.306884765625, 0.64697265625, 1.4345703125, -1.4736328125, 0.83935546875, -0.248779296875, 0.322998046875, -0.201416015625, 1.7734375, 0.2437744140625, -0.2021484375, -0.27197265625, -1.0390625, 1.0283203125, 1.9580078125, 0.658203125, 0.783203125, -1.3671875, -0.21484375, 0.3896484375, 0.063232421875, -0.160888671875, -1.2197265625, 0.27978515625, -1.7783203125, -0.404296875, -1.13671875, -1.3232421875, -0.7275390625, 1.4228515625, -0.6796875, -0.1524658203125, -1.4052734375, 0.26416015625, 0.1583251953125, 0.97900390625, -0.689453125, 0.701171875, -1.0556640625, 1.2509765625, -0.86572265625, 0.301513671875, 1.5205078125, 1.4345703125, 0.90673828125, -0.6298828125, -0.0400390625, -0.544921875, -0.79931640625, -0.7119140625, -0.2158203125, -0.82177734375, -0.67724609375, -0.4404296875, 0.90966796875, 0.045074462890625, -0.72509765625, 0.1953125, 0.962890625, 0.81591796875, -1.458984375, -0.87890625, 0.427001953125, 1.2080078125, -1.1376953125, 0.6259765625, 0.08282470703125, 0.794921875, -0.00726318359375, -2.548828125, -1.9267578125, 0.035247802734375, -0.78564453125, -1.646484375, -0.08721923828125, 0.10443115234375, 1.5625, -0.404296875, -1.2763671875, -1.806640625, 0.58447265625, 0.424560546875, 0.4052734375, 0.99365234375, -0.60302734375, 1.0166015625, 0.75048828125, -0.2210693359375, 0.76708984375, 0.62744140625, 0.447998046875, -0.46728515625, -0.10467529296875, 0.30712890625, 1.5751953125, 0.60107421875, 0.4697265625, -0.285400390625, -0.472900390625, -0.264404296875, -1.6689453125, 1.353515625, -1.611328125, 0.41064453125, -1.5224609375, -1.0146484375, 0.53857421875, -0.7431640625, 0.00740814208984375, 0.420166015625, -0.25439453125, 2.12890625, -1.6005859375, 0.1798095703125, 1.255859375, 0.343505859375, 0.344970703125, 2.04296875, -1.4189453125, 1.529296875, 1.80859375, 1.3251953125, 1.037109375, -2.544921875, -1.0009765625, 0.76904296875, 0.28564453125, 1.6845703125, -0.2301025390625, 0.986328125, -0.62451171875, -0.10882568359375, 0.4833984375, -1.1806640625, -0.9306640625, 0.1253662109375, -0.436767578125, 0.1402587890625, -0.8876953125, -0.98486328125, 1.6982421875, 0.405517578125, -0.353271484375, 0.32568359375, -0.92041015625, -2.22265625, 0.34375, -1.248046875, 0.94482421875, 1.9169921875, -0.5986328125, 1.0966796875, -1.3515625, -0.0132293701171875, 1.6103515625, 1.0546875, 1.2275390625, -0.48193359375, -0.5146484375, -0.64599609375, 1.0, 1.130859375, -0.65771484375, 0.951171875, 0.0279541015625, 1.220703125, -0.1280517578125, 0.443603515625, -0.1951904296875, -0.437255859375, 1.3916015625, -1.9697265625, 1.9052734375, -1.72265625, -0.81298828125, -0.91064453125, 0.462646484375, -2.3828125, -0.68896484375, 2.2109375, 0.79931640625, -0.76318359375, -0.01303863525390625, -0.1907958984375, 0.11114501953125, -0.705078125, -0.703125, 1.3095703125, 0.4423828125, -0.1724853515625, 1.4443359375, -1.775390625, 0.892578125, 1.814453125, -0.461181640625, 1.0224609375, 0.376220703125, -0.239501953125, -1.01171875, 0.8349609375, -2.57421875, -1.1015625, -2.044921875, 0.061767578125, -1.5400390625, -2.205078125, 1.0234375, 0.182861328125, -1.6328125, 0.4111328125, 1.1357421875, 1.0830078125, 1.0947265625, -0.8125, -0.1005859375, 0.83447265625, -1.1611328125, -0.5244140625, -2.408203125, 1.15625, 1.990234375, 1.6728515625, 0.268310546875, 0.78466796875, 0.483642578125, 1.478515625, -1.3720703125, 1.4052734375, -0.6796875, 0.5048828125, -1.87109375, 1.974609375, -0.09149169921875, -1.2392578125, -0.64208984375, -1.1318359375, 0.335693359375, -0.9228515625, 1.0146484375, 0.75439453125, -0.46435546875, 1.2080078125, 0.128662109375, 1.6943359375, 0.0987548828125, -1.240234375, -1.255859375, 0.84814453125, -1.9140625, 0.60205078125, -0.0343017578125, -1.4755859375, -0.783203125, 0.8505859375, 0.434814453125, -1.27734375, -0.342041015625, 1.4580078125, -1.6484375, -0.87060546875, 1.572265625, 1.748046875, -0.92822265625, -1.6416015625, -0.13818359375, -0.6435546875, 0.76318359375, -1.2138671875, 1.66796875, 2.0625, -0.399658203125, 0.82177734375, -2.072265625, 0.20556640625, -2.328125, 0.48779296875, 0.3662109375, -0.39892578125, -1.673828125, 2.283203125, 1.8798828125, -0.03363037109375, -1.416015625, 1.58984375, 0.25537109375, -0.6953125, -3.759765625, -0.62939453125, -0.00769805908203125, -1.728515625, -1.4814453125, 1.4482421875, -1.14453125, -0.68017578125, -0.216064453125, 0.1597900390625, 1.900390625, 0.05145263671875, 1.025390625, 1.0498046875, -1.8916015625, -0.64453125, 0.495849609375, 0.362548828125, -0.685546875, -0.388427734375, -0.1868896484375, -0.182373046875, 0.57373046875, -1.2841796875, -0.83837890625, 1.1005859375, 0.315185546875, 0.56982421875, -1.34765625, 0.77001953125, -0.053802490234375, -0.1163330078125, 0.0200347900390625, -1.1552734375, 0.896484375, 0.67626953125, -1.4423828125, -1.4462890625, -2.859375, 0.2257080078125, -1.2626953125, -0.671875, -1.0283203125, -0.87939453125, -0.81640625, -0.6220703125, 0.9912109375, -1.630859375, -0.433837890625, 0.5380859375, 0.08770751953125, 0.1285400390625, -1.076171875, 1.0576171875, -1.42578125, 0.453369140625, -0.1007080078125, -0.7890625, 0.0704345703125, -2.505859375, -1.029296875, -1.72265625, -1.3359375, 1.5751953125, -0.5029296875, 1.037109375, -2.052734375, -0.496826171875, -1.06640625, 0.5537109375, -0.018035888671875, -1.486328125, 0.82763671875, 0.110107421875, 0.242919921875, 1.0068359375, -0.2056884765625, -0.43505859375, 0.9208984375, 1.802734375, -0.11639404296875, -0.072265625, 1.228515625, 0.9365234375, 0.039581298828125, 0.40576171875, 1.4208984375, -1.197265625, 0.5859375, -2.79296875, 1.4384765625, 0.3818359375, 0.401123046875, -0.0579833984375, -1.890625, -0.580078125, -1.017578125, -0.494384765625, 0.513671875, 0.182861328125, -0.6923828125, -0.2978515625, -1.1494140625, -0.5927734375, -0.7939453125, 0.170654296875, 0.94921875, 1.5478515625, -1.30859375, 0.56591796875, -0.048828125, 1.103515625, -0.27294921875, 0.56494140625, -0.2041015625, 0.890625, 0.6171875, -1.4248046875, -1.427734375, -0.193115234375, 0.249755859375, -0.244873046875, -2.53125, 0.2548828125, -0.29248046875, 1.064453125, -0.43603515625, 0.19189453125, -1.1572265625, -1.7314453125, 0.767578125, -0.51220703125, 1.4267578125, -0.57373046875, 1.4658203125, 0.0193023681640625, 0.1114501953125, 1.8349609375, 0.76513671875, 1.888671875, -0.404052734375, 0.06573486328125, 0.390380859375, -0.359130859375, 0.024871826171875, -2.427734375, 0.67919921875, -1.1953125, 0.1224365234375, -0.78662109375, -0.10546875, -1.5732421875, 1.2119140625, -1.0390625, -0.7685546875, -0.6513671875, -0.82470703125, 0.78955078125, 0.3984375, -0.08172607421875, 0.6005859375, -0.57373046875, -0.58544921875, -0.20703125, 0.5537109375, -0.5712890625, 0.67236328125, 0.3466796875, -0.1754150390625, 0.223388671875, -1.6513671875, -0.3310546875, 0.011566162109375, -0.93798828125, 1.09375, -0.397216796875, 0.68896484375, -0.022918701171875, 0.88330078125, -1.0625, 0.0283203125, -0.87744140625, -1.005859375, -0.81591796875, -1.22265625, 0.02276611328125, -1.19921875, 1.236328125, 1.591796875, 0.05157470703125, 0.380126953125, -0.642578125, 1.2529296875, 1.8017578125, 0.4443359375, 1.380859375, 1.29296875, -1.060546875, -0.1558837890625, 0.387939453125, -0.78369140625, 0.169677734375, 0.77099609375, 0.23291015625, 0.4873046875, 0.7216796875, -0.037078857421875, 0.26904296875, 0.26513671875, 1.3427734375, 1.931640625, 0.1217041015625, -0.63818359375, -2.658203125, 1.58984375, 0.5205078125, -0.92529296875, 1.6435546875, -0.72509765625, 1.640625, -0.369384765625, 0.0596923828125, 0.7001953125, -0.27294921875, 0.232177734375, 1.4169921875, 0.25, 0.732421875, -0.66357421875, 0.685546875, 0.919921875, 0.365478515625, 1.2470703125, 1.6591796875, 0.473876953125, -0.57080078125, 1.30078125, -0.2333984375, -1.6201171875, -2.044921875, 1.29296875, 1.2158203125, 1.556640625, -0.6982421875, 0.7236328125, 0.347900390625, 0.66015625, 0.9072265625, -0.06353759765625, -1.9423828125, 0.81982421875, -0.1617431640625, 1.75, 0.39208984375, -0.6298828125, -0.89111328125, 0.04705810546875, 0.360107421875, -1.70703125, -1.1533203125, -0.2861328125, -0.261962890625, -0.4091796875, -0.2142333984375, 0.52294921875, -0.11663818359375, -0.984375, -0.296875, 1.0595703125, -1.388671875, -1.9580078125, 0.10308837890625, 0.81884765625, -0.69677734375, -0.85888671875, 0.9248046875, -0.47509765625, -0.368896484375, -0.307861328125, -1.14453125, -0.36669921875, -0.71630859375, -0.06793212890625, 0.44091796875, 0.86572265625, -1.0029296875, 1.46875, 0.047576904296875, -1.6494140625, 0.775390625, -1.1611328125, 0.096435546875, -0.1961669921875, -0.689453125, -0.0584716796875, -2.00390625, -0.13037109375, -0.44482421875, -0.426025390625, 1.626953125, 1.271484375, -0.3681640625, -1.267578125, 1.1435546875, 0.326416015625, -0.08251953125, -0.2137451171875, 1.25, -1.71875, 1.3017578125, 0.576171875, -1.2568359375, -2.44140625, -1.248046875, -0.4306640625, 1.2490234375, 0.5849609375, -0.286865234375, -0.0618896484375, -1.390625, -1.2998046875, 0.4580078125, 0.340087890625, 0.97119140625, -0.04071044921875, 2.25, -0.931640625, 1.5595703125, -1.7548828125, 1.123046875, 0.8818359375, 0.5703125, 0.67626953125, -1.2353515625, 2.40625, 1.4287109375, -0.057769775390625, 0.8984375, 0.8037109375, 0.49658203125, 0.62060546875, 1.9775390625, 0.30029296875, 0.499267578125, 1.126953125, 1.3330078125, -0.8076171875, 0.69287109375, 2.263671875, 1.4521484375, -0.90966796875, 0.2330322265625, 0.57470703125, 1.5517578125, -1.6396484375, -0.93798828125, -0.439453125, -3.12109375, -0.2113037109375, -1.84375, 0.1556396484375, 1.1826171875, -0.7080078125, 0.42138671875, -0.2169189453125, -0.09381103515625, -0.6328125, 1.8271484375, 0.2139892578125, -0.11492919921875, -0.1802978515625, -1.091796875, 0.375, 0.168701171875, 0.0176239013671875, 1.3583984375, -1.546875, -0.6181640625, -0.38134765625, -0.1572265625, -0.85595703125, -1.3857421875, -0.166015625, -0.75, -0.4072265625, -0.4599609375, -0.7001953125, 0.181640625, 2.48046875, -1.5546875, 0.921875, -1.9697265625, 0.98095703125, 0.427978515625, 1.7607421875, -0.7001953125, 0.01605224609375, -0.327392578125, -2.33203125, 0.402587890625, 1.279296875, 0.194580078125, 0.92919921875, -0.042572021484375, 0.87451171875, -0.407470703125, -0.37841796875, -1.380859375, 1.599609375, 0.0386962890625, 0.74609375, 0.175048828125, -0.248779296875, -1.0908203125, -0.483642578125, -0.701171875, 0.05743408203125, -0.7861328125, -0.0830078125, -1.7822265625, 0.357177734375, 0.1500244140625, 2.134765625, 0.6240234375, -0.96728515625, -0.5537109375, 1.4736328125, -1.02734375, 1.400390625, 1.7470703125, 1.49609375, -1.1123046875, -0.171142578125, 0.91552734375, 0.5498046875, -0.298828125, 1.0068359375, -1.6904296875, -1.4951171875, -1.033203125, 2.736328125, 1.0869140625, -0.517578125, -0.398193359375, 1.53125, 1.458984375, 0.61474609375, -0.54248046875, -0.35205078125, -0.71337890625, -0.240234375, -0.1351318359375, -1.541015625, -0.1153564453125, 0.161865234375, 0.482666015625, 0.56298828125, -0.30419921875, -0.421142578125, -0.200927734375, 0.73486328125, -0.0165557861328125, 0.89306640625, 0.0672607421875, -0.34521484375, 0.732421875, 0.6728515625, -0.4716796875, 0.0828857421875, 0.7958984375, 0.72314453125, -1.4990234375, -1.0498046875, -0.47314453125, 0.83544921875, 0.90087890625, 1.0673828125, 0.05950927734375, -0.206787109375, -2.23828125, 0.59326171875, 2.294921875, -1.0322265625, -1.4677734375, 0.6083984375, 0.2734375, -1.3427734375, 1.1689453125, 0.2467041015625, -0.250244140625, 1.005859375, -0.321044921875, -0.07513427734375, 2.673828125, 0.490966796875, 1.5732421875, 0.638671875, 1.1015625, 0.806640625, -0.237548828125, -0.9921875, 1.1787109375, -1.3359375, -0.05023193359375, -1.0927734375, -0.552734375, 0.60693359375, 0.623046875, 0.0026607513427734375, 1.1474609375, -1.546875, 1.064453125, 1.9775390625, 1.4599609375, -1.4443359375, 0.14111328125, 1.8251953125, -1.5107421875, 0.2020263671875, -0.71337890625, -1.3955078125, -1.0166015625, -0.9921875, -0.8369140625, -0.55078125, 0.2073974609375, -0.8388671875, -0.92236328125, -1.404296875, -0.402587890625, -0.40478515625, -1.05859375, 1.6875, -0.19189453125, 0.56787109375, 0.418212890625, 0.05938720703125, -0.66552734375, -0.493896484375, 0.9931640625, 0.309814453125, -1.6904296875, -1.6357421875, -0.69384765625, -0.95703125, -1.3076171875, 1.248046875, -0.2137451171875, -0.264404296875, 0.9169921875, -0.09649658203125, 0.071533203125, 1.8212890625, 0.1490478515625, -0.30322265625, -0.638671875, -1.263671875, 2.0625, -0.732421875, 0.09222412109375, 1.01953125, 0.41845703125, 0.75390625, -1.1552734375, 2.45703125, 1.2099609375, -0.447998046875, -1.048828125, 0.9853515625, -2.1484375, -1.1455078125, 0.43212890625, 0.95361328125, 0.93017578125, 1.86328125, -0.85400390625, 0.468505859375, 0.89501953125, 1.228515625, -1.2060546875, 0.467041015625, 1.078125, -2.658203125, -0.25634765625, -1.0283203125, 1.9453125, -1.595703125, -0.2166748046875, -2.66796875, 1.0712890625, -1.150390625, 1.8671875, -0.73779296875, 0.5927734375, 0.3271484375, -0.0809326171875, -0.59228515625, 0.1939697265625, 0.1351318359375, 0.779296875, -0.97998046875, 0.038970947265625, 0.92431640625, -0.50146484375, 0.053070068359375, -1.2021484375, -0.5615234375, -0.646484375, -1.5849609375, -1.48046875, 1.310546875, -0.58154296875, 1.7001953125, -2.626953125, -0.0379638671875, -0.388671875, -1.0302734375, 0.64306640625, -0.29150390625, -1.384765625, 0.482177734375, -0.44140625, 0.478759765625, 0.69775390625, -0.66552734375, -0.02459716796875, 1.41796875, -1.01171875, -1.4423828125, -0.86328125, 0.314697265625, 0.260009765625, 0.34716796875, -0.69970703125, 0.60888671875, -0.94091796875, 0.0193023681640625, 1.4814453125, -0.43994140625, -1.0078125, -0.03173828125, -0.9736328125, -1.6650390625, -0.278564453125, -0.4765625, -0.423828125, 2.478515625, 0.6884765625, -0.48876953125, -0.412109375, 1.126953125, 0.1767578125, 0.8515625, 1.595703125, -0.36181640625, -0.6884765625, 1.8837890625, -0.3701171875, 0.22021484375, -0.005420684814453125, -0.8369140625, -1.3603515625, 0.9599609375, -0.6181640625, -1.9892578125, 0.7216796875, -0.305419921875, 0.152099609375, -0.12237548828125, 0.495849609375, 2.30859375, -2.580078125, 0.6220703125, 1.5380859375, 1.0, 0.1513671875, -2.04296875, 1.0849609375, 0.3271484375, 1.041015625, 0.76171875, 0.244384765625, 1.7021484375, 0.68408203125, 0.720703125, -0.37646484375, -0.04937744140625, -0.6865234375, -0.1624755859375, 0.055023193359375, 2.416015625, -0.424072265625, -0.9951171875, 0.354248046875, 2.931640625, -0.89208984375, 0.8271484375, -2.15625, 0.54638671875, -0.56298828125, 0.86181640625, 0.51171875, 0.59716796875, -0.6337890625, -0.06744384765625, -0.17333984375, -1.1025390625, -1.5830078125, -0.05987548828125, 0.474853515625, 0.5673828125, 1.4462890625, 0.409423828125, -0.400146484375, 2.6640625, -0.5400390625, 0.34228515625, 1.6943359375, -1.1357421875, 1.712890625, -0.40234375, -1.130859375, 0.982421875, 1.0166015625, 0.049957275390625, -0.82177734375, -0.244873046875, -0.30078125, -1.521484375, 0.215087890625, -1.0537109375, 0.2197265625, 0.2783203125, 0.257568359375, 0.4716796875, 0.8603515625, -0.033660888671875, -1.28515625, 1.4658203125, -0.65283203125, -0.302734375, 0.03466796875, 0.2509765625, 1.505859375, 0.9140625, 0.69921875, 0.71044921875, 0.60205078125, -0.2220458984375, 0.44140625, 0.26708984375, 0.172607421875, 0.5673828125, -0.69091796875, -0.48046875, 0.80712890625, -2.029296875, -0.385986328125, -1.1298828125, 2.5703125, 1.10546875, 1.5537109375, -0.258544921875, 0.280517578125, -0.2208251953125, 0.685546875, -1.79296875, 1.4765625, 0.68896484375, -1.388671875, 0.47802734375, 0.8115234375, 0.032440185546875, 1.052734375, -0.407958984375, -0.434326171875, -0.81689453125, -0.5, 0.156005859375, -0.50830078125, -0.83642578125, 0.2122802734375, 0.1634521484375, -0.1558837890625, 1.4755859375, 0.54833984375, -0.022186279296875, 0.65673828125, -1.384765625, -0.0219879150390625, -0.04107666015625, -0.42578125, 0.284423828125, 0.046142578125, -1.341796875, -1.0029296875, -0.0014410018920898438, 0.052642822265625, 0.322265625, 0.67529296875, -0.94873046875, 2.216796875, 0.1634521484375, -1.359375, -0.2230224609375, -1.3251953125, -0.188232421875, 0.0999755859375, 0.53662109375, 0.113525390625, -0.410400390625, 1.130859375, -0.70458984375, 0.049163818359375, -0.309814453125, -1.337890625, -0.275146484375, -0.6748046875, -1.5126953125, -0.06671142578125, -0.04498291015625, 0.91748046875, -0.1480712890625, 0.71337890625, -1.0263671875, 0.342041015625, 0.76953125, 0.94091796875, -0.65673828125, -1.322265625, -1.1015625, 0.75390625, -2.7265625, 0.04833984375, 1.3671875, -0.60009765625, 0.294189453125, -0.1588134765625, 0.58203125, -0.9990234375, -0.86767578125, 1.724609375, -1.712890625, 0.71044921875, 0.75732421875, -0.79541015625, 2.09375, -1.7138671875, 0.06103515625, 0.2491455078125, 0.350341796875, -0.37744140625, -1.48046875, 1.916015625, -0.59423828125, -1.1474609375, 0.90283203125, -2.7421875, -1.2216796875, -0.765625, 0.640625, 2.013671875, -0.469970703125, 0.25537109375, -0.0999755859375, 0.72802734375, -0.303955078125, -1.6923828125, -2.625, -0.9228515625, -0.64501953125, 0.970703125, 0.151611328125, 2.162109375, 0.309326171875, -0.70458984375, 0.1671142578125, 0.27490234375, 2.478515625, 1.0810546875, -0.27490234375, -0.03753662109375, -0.07476806640625, 0.06585693359375, 0.60302734375, -0.32421875, 0.9013671875, -0.348876953125, 0.74462890625, -0.2200927734375, 0.373291015625, -0.272705078125, -0.70556640625, 1.09375, -0.89794921875, 1.3203125, 1.6923828125, 0.8984375, -0.92626953125, 0.083251953125, -0.305908203125, -1.9716796875, -0.367431640625, 0.56494140625, -0.81396484375, -0.70361328125, -1.9033203125, -0.040435791015625, -1.0361328125, 0.114013671875, -0.8662109375, 0.30224609375, -0.72607421875, 0.61279296875, 0.6552734375, -1.30078125, -0.326904296875, 2.203125, 0.541015625, 0.95654296875, -1.048828125, 0.4580078125, 0.1666259765625, -0.048583984375, -0.66455078125, -0.203857421875, 1.2626953125, 1.75, 0.00527191162109375, 0.2039794921875, 0.53466796875, 0.818359375, -0.2403564453125, -0.139892578125, -0.55615234375, -0.08770751953125, 1.181640625, -1.0625, -1.4365234375, 0.8212890625, -0.109375, 0.56982421875, 0.405029296875, 1.6494140625, -0.297119140625, -0.70166015625, 0.8486328125, 0.68212890625, -0.841796875, -0.54345703125, -3.43359375, 0.1373291015625, 0.56396484375, 0.0953369140625, -1.7109375, 1.208984375, -0.4423828125, -0.134033203125, 1.7783203125, -1.29296875, -0.32861328125, -2.255859375, 1.1025390625, 0.7451171875, 1.77734375, 0.0209503173828125, 1.15234375, 1.1357421875, 1.4990234375, 1.3388671875, -1.3759765625, -0.8037109375, -0.2044677734375, -0.61669921875, 0.0789794921875, -0.18603515625, 0.73583984375, 0.72705078125, -0.697265625, -1.119140625, -0.24267578125, -2.189453125, 0.1876220703125, -1.140625, -0.1268310546875, 0.042327880859375, 0.412353515625, -0.68115234375, 0.3544921875, -0.5322265625, 0.630859375, -0.10968017578125, -1.0537109375, 1.2822265625, -0.66943359375, 1.1865234375, -0.56005859375, -0.85986328125, 1.98046875, 0.396484375, 0.6083984375, 0.52490234375, 0.9892578125, -0.64892578125, -0.21533203125, 0.69677734375, -0.279541015625, -0.027557373046875, 0.08563232421875, 2.23046875, -0.88916015625, -0.51904296875, 1.1806640625, 0.75439453125, 0.70556640625, -0.0227508544921875, 0.53857421875, -1.6796875, -1.03515625, -0.86962890625, -1.265625, -0.78515625, -0.1683349609375, -1.7001953125, 0.8046875, -0.0479736328125, -0.95068359375, -0.81005859375, -0.044219970703125, -1.5576171875, 0.66162109375, -0.171142578125, 0.81201171875, 0.239990234375, -0.50634765625, -0.22900390625, -1.220703125, -0.876953125, -1.09765625, 0.3984375, -0.84716796875, 0.095947265625, 0.7763671875, -2.328125, -0.587890625, 0.336181640625, -2.009765625, 0.220703125, -0.6064453125, -0.7236328125, 0.50732421875, -0.8056640625, 0.9423828125, -1.6982421875, -0.0307769775390625, 0.288330078125, -0.9853515625, -0.82373046875, 0.04388427734375, 0.496337890625, 0.900390625, 0.767578125, 1.23046875, -2.26953125, -0.0765380859375, 1.3505859375, -0.2415771484375, 0.1583251953125, -0.236083984375, -0.0254669189453125, -0.6787109375, -0.10467529296875, 2.36328125, 1.0537109375, 0.6083984375, -0.1705322265625, -0.459228515625, -0.061187744140625, 1.2314453125, -1.615234375, -0.418212890625, 1.455078125, 0.257080078125, -0.0096435546875, 1.015625, -0.09783935546875, -0.62548828125, 0.16162109375, -1.3974609375, -1.3564453125, 0.0256500244140625, -0.36767578125, -1.501953125, 1.7119140625, -0.57861328125, 1.361328125, -0.402099609375, -2.080078125, -1.14453125, -0.230712890625, -1.0615234375, -0.50634765625, -0.68798828125, 0.5302734375, 1.0107421875, -1.3154296875, 0.0151519775390625, -0.4921875, 0.45556640625, 0.6875, 0.355712890625, 0.371337890625, 1.2431640625, 1.0107421875, 0.19140625, 1.560546875, 1.08984375, 0.74267578125, -1.3662109375, -1.5986328125, 1.3251953125, -0.415771484375, -0.1295166015625, 0.54736328125, 0.449462890625, -0.57666015625, -0.5947265625, 0.2998046875, -2.078125, -1.11328125, 1.0703125, 1.9072265625, -0.1134033203125, 0.1937255859375, 2.416015625, 0.36572265625, 0.7451171875, -0.50634765625, 1.341796875, 0.6064453125, 0.66650390625, 1.4716796875, -0.5869140625, -0.802734375, 0.515625, 1.779296875, -0.525390625, 0.5615234375, -1.4619140625, 0.880859375, -0.53076171875, -1.1220703125, 9.775161743164062e-06, 0.9365234375, -0.58447265625, -0.486328125, -0.529296875, -0.8828125, 0.161865234375, -1.310546875, -0.1708984375, 1.2509765625, 0.06695556640625, 0.41748046875, 0.1473388671875, -2.49609375, -0.38134765625, -0.744140625, 0.41162109375, -0.80810546875, 0.281982421875, 0.6865234375, -0.9404296875, 1.7841796875, -1.125, 1.326171875, 0.5849609375, 2.71875, 2.287109375, 0.75390625, 0.72802734375, 1.93359375, 1.462890625, -0.1837158203125, -0.8984375, -0.91845703125, 0.17919921875, -0.428466796875, -0.452880859375, -0.75048828125, -0.6826171875, -0.54443359375, -2.212890625, -1.546875, -1.4619140625, 0.453857421875, -0.64501953125, -0.335693359375, -0.5595703125, -0.41845703125, 1.482421875, -1.0498046875, 0.412109375, 0.39306640625, 1.8427734375, 0.147705078125, 0.34326171875, -1.013671875, -0.206787109375, -1.173828125, 0.63134765625, -2.23046875, -1.837890625, -1.5048828125, -0.049407958984375, -1.0146484375, 1.818359375, -0.10406494140625, -0.8681640625, -0.328857421875, 0.80029296875, 0.0999755859375, -0.89111328125, -0.84716796875, -1.0947265625, -0.9609375, -0.005573272705078125, 0.7080078125, 0.44140625, 0.90771484375, 1.6015625, -0.1591796875, -1.166015625, 0.04949951171875, -0.994140625, -0.0849609375, 0.471923828125, -1.345703125, -0.0175323486328125, -0.18798828125, 1.8515625, -0.053314208984375, -0.41650390625, -0.10186767578125, 0.62890625, 0.90869140625, -0.1934814453125, 2.2734375, 0.10125732421875, 2.048828125, -1.181640625, 0.0157623291015625, 0.305419921875, -0.318359375, -1.568359375, -0.54736328125, 1.7294921875, 1.84375, 2.48828125, 0.5302734375, 0.412109375, -0.2186279296875, -0.394287109375, 0.3896484375, 0.64306640625, 0.73486328125, 1.259765625, 0.8837890625, 0.433349609375, 0.203369140625, -0.26611328125, 1.84375, 0.10296630859375, 1.732421875, 0.98876953125, 0.307373046875, 0.2042236328125, 0.08978271484375, 0.206298828125, -0.59765625, -0.76708984375, 1.3740234375, -1.6064453125, 1.5576171875, 0.8974609375, -0.31494140625, -0.91552734375, 1.57421875, -1.5634765625, -1.3720703125, -0.451416015625, -0.0289764404296875, -0.9921875, 0.430419921875, 1.498046875, -0.1605224609375, 0.5068359375, 0.82666015625, 0.442138671875, -1.4853515625, 0.767578125, 1.107421875, 1.328125, 1.49609375, 0.10296630859375, -1.3837890625, -0.8388671875, -0.42236328125, 0.379638671875, -0.99169921875, 0.5244140625, -0.841796875, -1.2939453125, -0.3515625, -0.962890625, 0.259033203125, 0.266845703125, -0.9169921875, 0.1343994140625, 0.55908203125, 0.189208984375, -0.281005859375, -0.2210693359375, 0.09393310546875, -0.7216796875, 0.0458984375, -0.1427001953125, 0.61572265625, 1.236328125, -1.6123046875, 0.486572265625, -0.56982421875, 0.56640625, 0.59423828125, 1.24609375, 0.280029296875, -1.1064453125, -0.29541015625, 0.462890625, 0.58740234375, -1.3701171875, -0.169921875, -0.67626953125, 1.22265625, -0.94677734375, -1.2333984375, -0.2222900390625, 1.6865234375, -0.481201171875, 1.4775390625, 1.4326171875, 0.242431640625, -0.9892578125, -0.2310791015625, -0.73583984375, -0.26806640625, 0.7587890625, -1.5673828125, -0.6083984375, -1.0859375, -0.310546875, -0.4169921875, -2.044921875, -1.0859375, -0.03570556640625, -1.2861328125, -2.083984375, 1.2333984375, 0.94677734375, -0.459228515625, -0.5107421875, 1.2294921875, 0.10052490234375, 0.6943359375, 1.115234375, -0.383056640625, -0.70947265625, 0.76220703125, 0.9697265625, 0.833984375, 0.06048583984375, 1.5185546875, 1.33203125, -0.8388671875, 0.9482421875, 0.5068359375, 0.58154296875, 0.4658203125, 0.04449462890625, 1.193359375, -1.10546875, -0.37841796875, 1.0029296875, 0.2296142578125, 0.50830078125, -0.431884765625, 0.029876708984375, -0.498779296875, -0.609375, -0.19384765625, 0.90234375, -0.81591796875, 0.34033203125, -0.93408203125, -2.201171875, -0.8173828125, 0.2291259765625, -0.87353515625, -1.8427734375, -0.1544189453125, -0.85595703125, -0.646484375, -0.07318115234375, 1.5224609375, 0.5986328125, -1.08984375, -0.1365966796875, -0.65283203125, 0.46484375, -0.24365234375, 0.58837890625, -0.544921875, 1.20703125, -2.10546875, -1.3115234375, 1.201171875, 0.93408203125, -0.30078125, 1.259765625, -1.2001953125, 0.40771484375, -1.240234375, 1.5859375, 0.162353515625, -1.9296875, 1.294921875, 0.445556640625, 0.1336669921875, 0.82421875, -0.947265625, 1.5439453125, -0.1455078125, -0.05010986328125, -0.96240234375, 1.2041015625, 0.126220703125, -0.62744140625, -0.6826171875, 1.390625, -1.1455078125, -1.125, -0.430419921875, 0.6162109375, -0.669921875, -0.263671875, 2.099609375, 3.0, -1.87890625, 1.6865234375, 0.2044677734375, -0.65087890625, -0.411376953125, 0.1903076171875, -0.97802734375, 0.045440673828125, -0.2147216796875, -0.351318359375, 0.07086181640625, 0.1981201171875, 0.8359375, -0.2203369140625, 0.370849609375, -0.32275390625, -0.91357421875, -0.6953125, -1.828125, 1.1142578125, -0.29833984375, 2.763671875, 0.1558837890625, -0.4833984375, -1.1943359375, -0.9580078125, 0.94287109375, -0.59326171875, -0.87451171875, -0.7138671875, -0.5283203125, 0.35888671875, -0.0889892578125, -0.80517578125, 1.3759765625, -0.5869140625, -1.76953125, -0.64453125, 0.12841796875, -0.7568359375, -0.495361328125, 0.0009927749633789062, -0.646484375, 0.38427734375, 0.7841796875, 1.36328125, -1.7880859375, 2.052734375, -2.322265625, 0.257080078125, 0.82763671875, -0.049285888671875, 0.9638671875, 1.966796875, 0.630859375, -1.89453125, -1.1943359375, 0.45703125, -0.47412109375, 0.875, -0.69287109375, 0.34033203125, 3.0625, -0.25634765625, 0.58447265625, -1.6552734375, -2.1640625, -1.3740234375, -1.1787109375, 1.3837890625, -1.3359375, 0.30908203125, 0.67431640625, 0.8056640625, -0.87646484375, -1.228515625, 0.2320556640625, 0.2467041015625, 0.5888671875, -0.05706787109375, -0.06976318359375, 2.806640625, -1.193359375, -0.97265625, -0.346923828125, 0.87109375, 0.40283203125, 0.06439208984375, -1.1708984375, 1.470703125, -1.490234375, 0.6904296875, 0.93310546875, 1.7646484375, 1.791015625, 0.505859375, -1.5390625, 1.0400390625, -1.0537109375, 0.84326171875, -0.62060546875, 2.5390625, 0.49658203125, 1.009765625, -0.17529296875, -0.08154296875, -0.093994140625, -0.89892578125, 0.7001953125, 0.06500244140625, 1.0234375, 0.8662109375, 0.8486328125, -0.50244140625, -1.115234375, -1.1201171875, -0.365478515625, 0.00696563720703125, 0.004711151123046875, 0.0667724609375, 0.0166168212890625, -1.9912109375, -2.08984375, 0.1500244140625, 1.986328125, -0.53369140625, 0.12445068359375, -0.440185546875, 0.2822265625, -0.53955078125, 1.623046875, -1.306640625, -0.08538818359375, -0.0094451904296875, 0.93310546875, 1.1943359375, 1.1337890625, 2.220703125, -0.828125, -0.2437744140625, -1.8115234375, 0.086181640625, -0.15625, -0.47412109375, 0.45654296875, -0.188232421875, 1.291015625, -0.318603515625, 1.275390625, 0.313720703125, -1.3779296875, -0.26513671875, 0.321533203125, 1.5166015625, -0.66357421875, -0.441162109375, -1.0908203125, -0.5693359375, -0.29541015625, -0.053131103515625, 0.5009765625, -0.10736083984375, -0.469482421875, -0.331787109375, 1.4033203125, 0.490966796875, -0.145263671875, 0.6201171875, 0.03875732421875, 1.0810546875, -1.2294921875, 0.55078125, -0.089111328125, 0.00586700439453125, 0.82177734375, 0.4580078125, -0.57470703125, -0.42236328125, -1.087890625, -0.042724609375, 0.66845703125, -0.1337890625, 0.89306640625, -0.708984375, 0.77197265625, 2.005859375, -1.3388671875, 1.259765625, -0.381103515625, -0.31591796875, -0.57568359375, 0.75244140625, -0.2296142578125, -1.1171875, 1.7548828125, 1.3046875, -0.6044921875, 2.212890625, 1.0400390625, 1.1728515625, 1.0537109375, -0.5966796875, 1.1552734375, -0.223388671875, -0.2242431640625, 0.370361328125, -0.231689453125, 0.51123046875, -0.4267578125, -1.4580078125, 1.091796875, 0.99658203125, 0.289306640625, 0.41650390625, 0.89453125, 0.6220703125, 0.97265625, -1.072265625, 2.146484375, 2.4375, -0.444580078125, 0.350341796875, 1.5966796875, 1.1064453125, 1.599609375, -0.72119140625, -1.064453125, 0.66552734375, 0.486572265625, -0.25048828125, 1.4765625, -1.458984375, -0.68896484375, 0.81201171875, -0.873046875, -0.5078125, -0.25634765625, 2.1875, -0.178955078125, 0.73046875, 0.88525390625, 2.1875, -3.265625, 0.64453125, 0.40869140625, 0.6220703125, -1.3916015625, -0.505859375, -1.2685546875, 1.4921875, 1.8369140625, 0.60498046875, 0.10784912109375, -0.978515625, 0.5146484375, 0.96923828125, 0.462646484375, 0.67724609375, -1.349609375, 0.53466796875, 2.091796875, 1.244140625, -0.294921875, -0.005340576171875, -1.0791015625, 1.1953125, 0.256103515625, 1.1962890625, -0.264404296875, -0.2305908203125, -0.57177734375, 1.0029296875, 0.84716796875, -0.478759765625, 0.55517578125, -0.580078125, 2.9140625, -0.389404296875, 1.669921875, -1.4326171875, 1.4853515625, -0.283935546875, 0.1461181640625, -0.06640625, -0.0290374755859375, 0.8251953125, 1.912109375, 1.8046875, 0.213134765625, -0.54052734375, 1.998046875, 0.8212890625, 1.4658203125, -1.82421875, 0.493408203125, -0.099365234375, 0.426025390625, -1.69140625, 0.462890625, 0.94482421875, 0.02325439453125, 1.1884765625, 2.029296875, -1.0234375, -0.95556640625, -1.43359375, -2.326171875, 1.0390625, 1.306640625, -0.8720703125, 0.990234375, 0.93505859375, -0.1409912109375, 0.17919921875, 0.43994140625, -0.62353515625, 0.62353515625, 0.84375, -1.4990234375, 0.4228515625, 1.86328125, 0.63623046875, 0.55859375, 0.951171875, 1.3134765625, 1.2587890625, 0.56494140625, 1.181640625, -0.81494140625, 0.03448486328125, 0.247802734375, 1.3564453125, 1.5947265625, 1.1669921875, 1.150390625, 0.966796875, 0.326171875, 0.02069091796875, 0.56689453125, 0.75634765625, -0.76513671875, -0.71142578125, 2.041015625, 0.12286376953125, -1.1484375, 1.6015625, -1.115234375, -0.80224609375, -0.54296875, -0.50341796875, 1.111328125, -0.2493896484375, 1.4736328125, 0.9189453125, -1.1416015625, -0.4814453125, -0.490966796875, 1.123046875, 1.080078125, 1.4541015625, 0.6728515625, -0.68603515625, 1.099609375, -0.366455078125, -0.908203125, -1.513671875, -0.65478515625, 0.4130859375, -0.751953125, -1.1240234375, 1.091796875, 1.2939453125, -0.4521484375, -1.4423828125, -0.0391845703125, -0.67724609375, -1.4443359375, 1.3994140625, 0.07159423828125, 1.1953125, -0.425048828125, 0.90185546875, 1.1708984375, 0.9560546875, -0.7822265625, -0.48291015625, -1.2958984375, -0.192138671875, -1.0517578125, 1.1103515625, -0.275634765625, 0.288330078125, -2.513671875, 0.03662109375, -0.607421875, 0.5810546875, 0.29296875, -0.84375, -1.1640625, -1.314453125, 0.3427734375, 0.71044921875, 0.53759765625, 0.44287109375, -0.1973876953125, 0.264404296875, 0.05206298828125, -0.857421875, 0.6025390625, 0.08203125, -0.040191650390625, 0.759765625, 1.013671875, 0.48583984375, -0.77978515625, -1.546875, -0.0103607177734375, 1.0478515625, -1.083984375, 0.041534423828125, -0.28466796875, 1.193359375, -1.06640625, -1.3173828125, 0.406982421875, 1.9189453125, 0.79736328125, -0.986328125, 1.5400390625, 1.212890625, -0.0849609375, -0.304931640625, 1.7763671875, -0.0267333984375, 0.01629638671875, -0.8828125, 0.89501953125, -0.1533203125, 0.80517578125, 0.5009765625, 0.86279296875, -0.6826171875, -1.2587890625, 1.33984375, 1.0302734375, -2.1484375, -0.31689453125, 0.4912109375, 0.8056640625, -0.892578125, -0.66845703125, 0.2099609375, 0.04180908203125, -0.810546875, -0.0535888671875, -0.4140625, -0.72412109375, -0.1583251953125, 0.95361328125, -1.58203125, 1.5078125, -0.06494140625, -1.2138671875, -1.779296875, 0.75244140625, 0.74658203125, -1.541015625, -0.7216796875, -0.798828125, 1.9375, -0.81005859375, 1.1298828125, 0.708984375, 0.40478515625, -1.216796875, 0.69580078125, 0.07470703125, 1.2626953125, -1.150390625, -1.5771484375, -0.354736328125, 0.317138671875, 1.328125, 0.038055419921875, 0.352783203125, 0.04827880859375, -0.0694580078125, -0.84814453125, 0.05560302734375, -1.02734375, -2.3125, -0.19384765625, 0.1522216796875, 0.74365234375, -0.2318115234375, 0.85986328125, -0.85498046875, -0.362548828125, 1.033203125, -0.53466796875, 0.28369140625, -0.32666015625, -1.21875, -0.02130126953125, -1.060546875, 0.80419921875, 0.48095703125, 0.50732421875, 1.9521484375, -1.0986328125, -0.763671875, 1.203125, 0.05975341796875, -0.75048828125, -0.728515625, -1.9619140625, -0.8779296875, 1.376953125, 0.057769775390625, -0.79541015625, 0.326904296875, -1.39453125, 1.615234375, -0.765625, -0.493896484375, -0.225830078125, -1.18359375, -0.082763671875, -0.71435546875, -0.7978515625, -0.158203125, 1.8525390625, -1.0478515625, 0.150634765625, -0.41552734375, 0.205810546875, -0.49462890625, -0.467041015625, -1.890625, -0.1729736328125, 0.467529296875, -0.408447265625, 0.77783203125, -0.32177734375, -0.34375, -1.7529296875, 1.26171875, -0.382080078125, -0.6552734375, 0.137939453125, -1.095703125, 0.5517578125, 2.24609375, -0.91943359375, 0.127685546875, 0.4189453125, 0.66650390625, 0.869140625, 0.060577392578125, -0.08306884765625, -0.5439453125, 1.482421875, -0.79345703125, -0.42431640625, -0.56396484375, -1.6875, -1.61328125, -1.96484375, -1.259765625, -0.391845703125, -0.58203125, 2.162109375, -1.1923828125, 0.74609375, -1.1513671875, 0.469970703125, 0.1260986328125, 0.0287933349609375, 1.3388671875, -0.58056640625, -0.64501953125, -0.059661865234375, 1.1650390625, 0.88818359375, 0.46533203125, -1.63671875, 0.412109375, -0.84228515625, -0.62353515625, 0.619140625, -0.09417724609375, 1.181640625, -1.4501953125, 0.71875, -0.07659912109375, 1.3505859375, 0.0303497314453125, -0.7939453125, -1.05078125, -0.802734375, 0.373291015625, -0.05078125, -1.388671875, -1.3876953125, 0.055084228515625, -0.52392578125, -1.05078125, -1.302734375, -0.1466064453125, 0.270263671875, -1.90234375, 1.04296875, 0.58544921875, -1.4287109375, -0.92138671875, 2.05859375, -0.2452392578125, -0.197509765625, -0.85009765625, -1.3212890625, 0.1343994140625, -0.1741943359375, 0.0220489501953125, 2.71484375, -0.49951171875, -0.1544189453125, -0.8369140625, -0.441162109375, -0.56298828125, 1.0, -1.41015625, 1.5205078125, 2.40234375, -1.615234375, -1.455078125, 0.89697265625, 1.68359375, 0.9814453125, 1.1142578125, 0.52197265625, -0.054901123046875, -0.59521484375, 0.052734375, 2.30859375, -1.48828125, -0.97119140625, 1.455078125, 0.72021484375, -0.88525390625, 0.01136016845703125, 1.54296875, -0.246337890625, -1.0302734375, 0.83935546875, 0.93115234375, 1.56640625, 1.68359375, 0.322998046875, 1.5810546875, -1.9931640625, 0.44775390625, 0.3779296875, -1.40625, -0.76025390625, 1.38671875, 0.90478515625, 0.463623046875, 0.67626953125, -0.50439453125, -1.1435546875, -0.19970703125, 0.6376953125, 0.61181640625, -1.7646484375, -1.80859375, -0.37255859375, 0.810546875, 0.361572265625, 0.76611328125, 0.2225341796875, 1.302734375, -0.189208984375, 0.166015625, -0.08160400390625, 0.396240234375, 0.37060546875, -0.5595703125, -0.84228515625, -0.62255859375, -1.1103515625, -0.8974609375, 0.74267578125, 0.384033203125, -1.078125, -0.0645751953125, 0.12237548828125, -0.591796875, 2.666015625, 0.69580078125, -0.34619140625, 1.166015625, -0.68017578125, -0.03863525390625, 0.24072265625, 0.66357421875, -1.3837890625, 1.001953125, 0.11456298828125, -0.78271484375, -0.97900390625, -1.607421875, 0.1229248046875, 0.681640625, -0.537109375, 1.30859375, 0.642578125, -0.2030029296875, 1.1533203125, -1.4560546875, 0.6513671875, -0.037872314453125, 1.1396484375, -0.5009765625, -0.57763671875, -1.9501953125, 1.4990234375, 0.2646484375, -0.0328369140625, -0.1937255859375, -0.1494140625, -0.947265625, 0.59326171875, 0.88232421875, 1.2880859375, 0.203369140625, 1.5888671875, 0.150634765625, 0.6103515625, -1.1123046875, 0.176025390625, -1.1171875, 0.81982421875, -0.1641845703125, -1.3818359375, 1.076171875, 1.3173828125, -0.1649169921875, -0.268798828125, 1.18359375, 0.51953125, -3.1015625, 0.6494140625, -1.380859375, 0.5732421875, -1.35546875, 0.08441162109375, 0.07427978515625, 1.380859375, 2.337890625, 0.01145172119140625, 0.1876220703125, 0.02459716796875, 0.845703125, -1.8671875, 0.40380859375, 0.356689453125, 0.1939697265625, 2.708984375, -0.2900390625, -0.1629638671875, -0.53369140625, -1.0810546875, -1.064453125, 0.10003662109375, 0.002437591552734375, -0.13623046875, 0.61083984375, -0.1644287109375, -1.80859375, 1.025390625, -0.262451171875, 1.01953125, 0.3828125, -1.345703125, 0.76416015625, -0.8603515625, 0.9169921875, -1.888671875, 1.2158203125, 0.5087890625, 2.708984375, -1.3095703125, -0.552734375, 0.317626953125, 0.200439453125, -0.720703125, 0.489990234375, -0.4326171875, -0.333740234375, -0.927734375, -2.015625, -0.49462890625, 1.0654296875, -0.398681640625, 1.1015625, -0.8193359375, 0.5517578125, 0.31982421875, 3.421875, -1.33203125, -0.009552001953125, -0.348388671875, -0.9111328125, -0.78515625, 1.123046875, -1.5634765625, -0.3408203125, 2.61328125, 0.1302490234375, 1.236328125, -0.4794921875, -1.177734375, -0.09991455078125, 0.012725830078125, -2.203125, 1.76171875, 0.77978515625, 0.1435546875, 0.8271484375, -0.306884765625, -1.25390625, -0.4013671875, 1.009765625, -0.328125, -2.001953125, 0.393310546875, -0.7294921875, 0.356201171875, -0.02978515625, -2.12109375, -0.83935546875, -0.284423828125, 1.427734375, -1.0634765625, -0.2802734375, 1.474609375, -0.09271240234375, 0.896484375, -1.0224609375, 0.0246124267578125, 0.34326171875, 0.7373046875, -0.736328125, 0.990234375, 0.0572509765625, -0.72265625, 0.92041015625, 1.990234375, -0.8212890625, -0.943359375, 0.685546875, -0.144775390625, -0.205810546875, 0.92041015625, 0.48828125, 0.6201171875, 2.119140625, 1.8779296875, 1.3115234375, -0.66357421875, -0.58154296875, 1.6103515625, 1.3017578125, -0.0643310546875, -0.76953125, -1.435546875, -0.5947265625, 1.5732421875, 0.0186004638671875, -0.45166015625, 0.83544921875, 0.294189453125, -0.434814453125, 0.489990234375, 0.438720703125, -2.416015625, 1.30078125, -1.1064453125, -0.060028076171875, 0.92431640625, -0.460205078125, 0.6005859375, -0.53076171875, -0.42529296875, -0.93359375, -0.441650390625, 0.2479248046875, -0.62353515625, -1.8505859375, -0.369384765625, -0.343505859375, -2.03125, -0.39404296875, 1.1806640625, 0.1357421875, 1.0712890625, -1.2392578125, 0.73095703125, -0.36572265625, -1.5234375, 0.0738525390625, -0.103759765625, -1.3046875, 0.357421875, -0.167236328125, 1.4765625, -1.7626953125, -1.04296875, -1.1318359375, -0.2626953125, -0.95263671875, -0.020233154296875, -0.385009765625, -1.3779296875, -2.87890625, -0.0257720947265625, -1.064453125, -0.2127685546875, 0.69287109375, -0.18798828125, 0.60546875, -1.4091796875, -0.78076171875, -0.67724609375, -0.81640625, -1.44140625, 0.331787109375, 0.1412353515625, 1.044921875, -0.11859130859375, -0.39697265625, 0.8564453125, -0.428466796875, 0.2861328125, -0.986328125, 1.078125, 0.40380859375, 0.3515625, -1.205078125, -0.701171875, 0.383056640625, 0.8466796875, 0.67529296875, 1.171875, -0.58251953125, 0.6337890625, -0.76025390625, -0.333984375, -0.382568359375, -1.4501953125, 1.1181640625, 0.72314453125, -0.3603515625, -0.58251953125, 1.1513671875, 0.64990234375, 2.08203125, 0.32275390625, -0.01187896728515625, -0.1588134765625, -1.9462890625, -0.72216796875, -0.92724609375, 0.442138671875, 1.107421875, -0.13818359375, 1.21484375, 0.53564453125, 0.61376953125, 0.76953125, -1.98828125, 0.6650390625, 0.1610107421875, 0.490478515625, 0.294189453125, -1.5498046875, -0.50048828125, 0.11956787109375, 1.814453125, -0.43212890625, -1.30078125, 0.354248046875, -0.148193359375, -0.3232421875, -1.5478515625, -2.205078125, 0.46484375, 0.2244873046875, -0.215087890625, 1.5478515625, -0.461181640625, 1.134765625, -1.083984375, -0.67333984375, 2.5, -0.404052734375, 0.131591796875, -0.666015625, -1.15625, 0.5009765625, 0.470947265625, 0.78369140625, -0.96923828125, -0.95849609375, 0.4248046875, 0.439208984375, 0.4287109375, -0.40673828125, 0.607421875, 0.432373046875, -0.88525390625, 0.79296875, -1.453125, 0.56494140625, -0.039459228515625, 0.40478515625, 1.0302734375, 0.97119140625, -1.130859375, 1.2470703125, -0.350830078125, -1.0537109375, 0.19970703125, -0.4775390625, 0.2476806640625, 0.06787109375, 0.47998046875, 0.02642822265625, 1.626953125, -0.333984375, -1.537109375, -0.427978515625, -0.0728759765625, 1.0400390625, 1.3896484375, 0.09942626953125, -0.23876953125, -0.9609375, -0.85546875, 2.283203125, -0.29931640625, -0.896484375, 0.56201171875, -2.2578125, -1.876953125, 1.0576171875, 0.3798828125, 0.25341796875, -0.0687255859375, -0.6240234375, 1.857421875, -1.5703125, -0.20654296875, -0.318603515625, 0.3525390625, 0.25244140625, 0.46875, 2.005859375, -0.1236572265625, 1.009765625, -2.75390625, -1.470703125, -0.81396484375, -0.482421875, -1.03515625, -0.3134765625, -0.470947265625, -0.345947265625, 1.380859375, -0.0101165771484375, 0.455078125, 1.9873046875, 0.71533203125, -0.78125, 0.36669921875, -1.8662109375, -0.98974609375, 0.62353515625, -0.09088134765625, -0.09320068359375, 3.359375, 1.802734375, 1.4453125, -0.8603515625, 0.5830078125, 0.1436767578125, -0.1553955078125, 0.65869140625, -0.5146484375, 0.6708984375, -0.60888671875, -0.404052734375, 0.1328125, -0.28857421875, -0.182373046875, 0.361328125, -0.2861328125, 0.89697265625, 0.98583984375, -0.497802734375, -1.3828125, 0.0172576904296875, 0.1302490234375, -0.241943359375, -0.77783203125, 0.78564453125, -0.97314453125, 2.775390625, 1.765625, -2.318359375, -1.4130859375, 1.0078125, 1.427734375, 0.5537109375, 0.54296875, 1.1728515625, -1.0087890625, -0.69580078125, -0.49951171875, -1.0302734375, 0.059783935546875, 0.572265625, 0.375244140625, -0.81787109375, 1.794921875, -0.44384765625, -0.1279296875, -0.4853515625, 1.908203125, 0.4072265625, -0.359619140625, -0.4775390625, 0.34423828125, 0.67529296875, 0.1063232421875, 1.140625, 1.7626953125, -0.91943359375, -0.481201171875, 0.2410888671875, 0.43115234375, -0.271240234375, 0.275634765625, 0.317138671875, 0.394287109375, -0.74169921875, 0.8818359375, 0.6640625, 1.615234375, -0.92626953125, -0.9873046875, 1.6103515625, -0.4462890625, -0.424072265625, 0.27587890625, 0.12310791015625, 0.38525390625, -1.494140625, -0.91455078125, -0.564453125, 0.5732421875, -2.28515625, -2.54296875, 1.0634765625, 0.572265625, -0.0797119140625, 0.9384765625, 1.357421875, 0.317138671875, -0.751953125, -1.0859375, 0.515625, 1.3095703125, -0.28173828125, 1.10546875, -0.701171875, 0.2998046875, 0.3046875, -2.580078125, -0.384521484375, 0.356689453125, 0.303466796875, -0.85986328125, -0.55517578125, -0.484130859375, 0.1588134765625, 0.51708984375, -1.2421875, -1.107421875, -1.056640625, -0.11517333984375, 1.234375, -1.1337890625, -1.4814453125, -0.89306640625, -0.53076171875, 0.6513671875, 1.58203125, 0.389404296875, -1.7158203125, -0.46533203125, -0.4521484375, 1.349609375, -0.50146484375, 1.490234375, 2.083984375, 2.306640625, -0.66015625, -0.77880859375, -1.01953125, -0.607421875, 0.87060546875, -0.062286376953125, -0.0211181640625, 1.71484375, 1.8115234375, 0.369873046875, 1.1640625, 0.2081298828125, 1.162109375, -0.2117919921875, -0.69384765625, -2.236328125, 0.98876953125, -2.345703125, 0.451171875, -0.2037353515625, 1.6279296875, 0.88671875, -0.60302734375, 0.68701171875, -0.9423828125, 0.2822265625, 0.01419830322265625, 0.397216796875, -0.071044921875, 0.09075927734375, -0.416015625, -0.9521484375, -0.8310546875, -0.87451171875, 0.2113037109375, 0.07440185546875, -0.174072265625, -0.305908203125, -1.1708984375, 0.953125, 0.55712890625, -0.03692626953125, -0.67529296875, -1.3564453125, -1.8759765625, 0.61962890625, 0.0259552001953125, -0.1693115234375, 0.421142578125, -0.3779296875, -0.51904296875, -0.47314453125, 0.77978515625, -0.272705078125, -1.02734375, 1.5908203125, 0.221435546875, 0.751953125, 0.44384765625, 0.1724853515625, -0.18408203125, 1.009765625, 0.1983642578125, 1.728515625, 2.609375, -0.191162109375, 0.9443359375, -0.63525390625, 0.287109375, -0.7275390625, 0.136962890625, -0.880859375, -0.1890869140625, -0.19287109375, 0.33203125, 0.07318115234375, -1.38671875, 1.8271484375, 0.11199951171875, -0.49658203125, -0.166015625, 0.4345703125, -0.8603515625, 0.68310546875, 1.1513671875, -1.2900390625, -1.111328125, -0.984375, 1.29296875, -1.4462890625, 0.09320068359375, -0.275146484375, 0.12469482421875, -0.703125, 1.4365234375, -0.357421875, -1.3759765625, -0.5087890625, -0.093994140625, 0.0386962890625, 0.026702880859375, -0.62353515625, -0.58203125, -0.486572265625, -1.34375, -0.3134765625, -0.047454833984375, 1.5478515625, 0.283203125, 0.91845703125, -0.71484375, 1.0732421875, 1.5380859375, 1.1162109375, -0.73779296875, 0.09417724609375, -0.1434326171875, 0.21484375, 0.050750732421875, -0.1998291015625, -1.59765625, 2.330078125, 0.455810546875, -0.603515625, -1.490234375, -0.349853515625, 0.1663818359375, -0.468505859375, 0.4794921875, -0.6787109375, 1.609375, -2.080078125, 1.1806640625, -0.189453125, 1.482421875, 0.50390625, -0.53564453125, -0.1417236328125, -0.70849609375, 0.755859375, 0.49462890625, 0.85888671875, 0.59033203125, 0.5546875, -1.4716796875, -1.5224609375, 0.96630859375, -1.2607421875, 0.08074951171875, 0.462646484375, -0.45166015625, -0.06390380859375, 0.1328125, 0.2060546875, 1.98046875, 1.228515625, 1.1748046875, 0.427978515625, 1.2080078125, -0.92529296875, 0.29736328125, 1.4912109375, -1.6962890625, -0.97216796875, -0.060272216796875, -0.59716796875, -2.001953125, 1.6015625, 0.06280517578125, -0.75244140625, 0.537109375, -1.57421875, -1.7880859375, -0.48046875, -0.70751953125, 1.623046875, -0.89501953125, 0.323486328125, 0.037384033203125, -1.1435546875, -0.152099609375, 0.2088623046875, -1.67578125, -0.6005859375, 0.60107421875, -0.189697265625, 0.2073974609375, 1.2197265625, -0.404052734375, -1.6396484375, -1.25390625, 0.1837158203125, 0.373291015625, 1.5068359375, -1.3818359375, -0.019073486328125, -1.171875, -1.2080078125, 1.7119140625, 0.94140625, -0.7314453125, -0.05609130859375, -0.38330078125, -0.480712890625, -1.4443359375, 0.01425933837890625, -1.5400390625, -0.49560546875, 0.5908203125, 1.3935546875, 1.05078125, 0.55810546875, -0.455078125, -1.0556640625, -0.4345703125, -0.5185546875, 1.9013671875, 0.94091796875, 0.0225830078125, -0.6748046875, -1.875, 0.3427734375, -1.8681640625, 0.1219482421875, 0.1302490234375, -1.3955078125, 1.97265625, -0.2220458984375, 1.732421875, 0.853515625, -0.552734375, 0.0164794921875, 0.53515625, -0.9052734375, -0.313232421875, -1.271484375, -1.4873046875, -0.765625, 0.55859375, -0.4814453125, -0.419921875, 1.115234375, -2.958984375, -0.236572265625, 0.7158203125, -0.1463623046875, 1.673828125, -0.77392578125, 1.9091796875, -0.1812744140625, 0.0662841796875, -0.35791015625, 0.2349853515625, -0.58935546875, 0.94384765625, -0.65380859375, 0.91650390625, 0.88720703125, 1.09375, -1.7060546875, -2.080078125, -0.23681640625, 0.212158203125, -0.217529296875, 0.44140625, -1.15625, 0.58740234375, -1.3955078125, -0.66162109375, -0.036224365234375, -1.5185546875, 1.1611328125, 0.67333984375, 0.12200927734375, 0.302001953125, 0.99072265625, 0.92919921875, -1.912109375, -0.55712890625, -1.00390625, -0.58740234375, -0.476318359375, -0.24755859375, -0.68505859375, 1.0078125, 0.4033203125, -0.1993408203125, -1.8125, 0.55126953125, -2.044921875, 1.42578125, -0.61865234375, -1.1416015625, -1.1904296875, 3.359375, -1.455078125, -0.1412353515625, 1.244140625, 0.8134765625, 1.9072265625, 0.1943359375, 1.1806640625, -0.433349609375, -1.859375, -0.371337890625, -0.8388671875, -0.413330078125, -0.8427734375, -0.5634765625, -0.58154296875, -0.497802734375, -0.09942626953125, 1.103515625, 1.52734375, -0.2230224609375, 0.26806640625, 0.432861328125, -1.091796875, 3.146484375, -1.740234375, -0.047393798828125, -0.732421875, 1.291015625, -0.65380859375, 0.448974609375, -1.6865234375, 0.7783203125, 0.84521484375, -2.083984375, -1.1669921875, 0.2047119140625, -1.8857421875, 0.5595703125, -0.281982421875, 0.94384765625, 2.6875, -0.316162109375, -0.07586669921875, -1.55859375, -0.046875, -1.099609375, 0.4345703125, 1.189453125, -0.2398681640625, 0.91015625, -0.50341796875, 1.2333984375, -0.424560546875, 1.1513671875, -0.9150390625, -0.19970703125, 0.09210205078125, 0.64208984375, 1.55078125, -0.6611328125, -1.146484375, 0.3515625, 0.1937255859375, 0.298828125, -1.43359375, 0.8251953125, -0.62646484375, 0.0167083740234375, -1.88671875, -2.15625, 1.7236328125, -0.5947265625, -0.58154296875, -1.3515625, -0.235107421875, 2.1875, -0.52734375, -1.45703125, -1.34765625, -0.047088623046875, -0.06365966796875, -0.9833984375, 0.76318359375, 0.385986328125, 0.2259521484375, -0.11187744140625, 0.441162109375, -0.0537109375, -0.9814453125, -0.361328125, 0.453369140625, 0.75341796875, -0.258544921875, 0.492919921875, -0.1395263671875, 0.1998291015625, 1.390625, 3.0, 1.0791015625, 0.55078125, 0.52392578125, -0.359619140625, -0.377685546875, -0.29150390625, 0.57470703125, -0.53662109375, -1.150390625, -0.59521484375, -0.029052734375, -2.1015625, -1.3349609375, 0.37744140625, -0.947265625, 0.59228515625, 0.40625, -0.77099609375, 0.83203125, -1.3828125, 0.313232421875, -0.5166015625, 0.70068359375, -0.5126953125, -2.033203125, 1.03125, 0.41943359375, 1.044921875, -1.12109375, -1.279296875, -0.024200439453125, -0.8173828125, 0.65234375, -0.468505859375, 0.0223541259765625, 0.31005859375, -1.23046875, -1.8427734375, -0.01788330078125, 0.407958984375, -1.4482421875, 0.47607421875, -0.483642578125, -0.96484375, -1.7705078125, -0.1395263671875, 0.374267578125, -0.357421875, -1.5537109375, -1.330078125, 0.923828125, 0.03436279296875, 0.2474365234375, 1.17578125, -0.609375, 0.509765625, -0.1947021484375, 1.1748046875, 0.6572265625, 1.412109375, -0.11370849609375, 0.5498046875, -0.004787445068359375, -1.2333984375, 0.6357421875, 0.449462890625, 2.595703125, 1.9755859375, -1.2392578125, 1.65625, -0.476806640625, -2.2109375, -0.377197265625, 0.203857421875, 1.81640625, 0.3681640625, 0.4287109375, 0.1444091796875, -0.39501953125, -0.9423828125, 0.08746337890625, -0.47412109375, -0.3681640625, 0.01178741455078125, 0.2020263671875, 0.71435546875, -0.0173187255859375, 0.021270751953125, -0.1375732421875, -0.499267578125, 0.8681640625, 0.1700439453125, -0.01296234130859375, 1.041015625, -1.099609375, -0.146484375, -1.89453125, 1.279296875, -0.3828125, -0.634765625, 1.0234375, 0.95703125, 0.1282958984375, -0.70849609375, -0.329833984375, -0.3544921875, -0.3662109375, 0.295166015625, -0.430419921875, 1.6767578125, -1.3701171875, -0.4833984375, -1.4580078125, 0.043609619140625, 0.69091796875, -0.29541015625, -1.8916015625, 0.5341796875, -1.4814453125, -1.2333984375, 0.1361083984375, 1.4140625, -0.411376953125, 0.75, -0.87451171875, 1.873046875, 0.054595947265625, -0.5263671875, 0.035400390625, 0.32666015625, -0.1258544921875, -0.072265625, 0.822265625, 0.90771484375, 0.01580810546875, -1.2236328125, 0.62451171875, -1.4794921875, -0.3017578125, 0.056304931640625, 0.333251953125, 0.373291015625, -0.1427001953125, 0.8046875, -0.0069122314453125, -0.63623046875, -0.458984375, 0.5439453125, 0.919921875, 0.5419921875, 0.014312744140625, 0.53564453125, 0.300537109375, 0.0367431640625, 0.6416015625, -0.051605224609375, -0.82275390625, -1.912109375, -0.4580078125, -0.0009765625, 0.20263671875, 0.64794921875, 0.222900390625, -0.138671875, 1.18359375, 0.0772705078125, 0.1575927734375, -1.0712890625, 0.2841796875, -1.2919921875, -0.6279296875, 0.8681640625, -1.30078125, -0.03814697265625, 1.04296875, -0.89208984375, 0.7880859375, -1.5732421875, -0.003635406494140625, -0.403076171875, 0.7421875, -1.076171875, 2.14453125, -0.97607421875, -0.367431640625, 0.6357421875, 0.485595703125, 0.80615234375, 0.71337890625, 0.3466796875, -0.1724853515625, -1.0361328125, -0.126953125, 0.3505859375, -0.9599609375, 0.146484375, -0.56591796875, 0.7373046875, -0.482421875, 0.31640625, 0.86962890625, 0.6220703125, -0.048095703125, 0.1763916015625, 0.214599609375, -0.455322265625, -0.66796875, 0.1507568359375, -0.78662109375, -0.06890869140625, 0.60205078125, 1.9248046875, -1.55078125, 0.78564453125, 0.82666015625, 0.298583984375, 0.51025390625, 0.36328125, 1.0283203125, -0.60595703125, -0.818359375, -0.88427734375, 0.90966796875, -0.52734375, 0.496337890625, 1.0615234375, 0.9755859375, 1.138671875, 0.77001953125, -0.282470703125, 0.85888671875, 0.63916015625, -1.2763671875, -0.27197265625, -0.08831787109375, -1.0693359375, 0.09771728515625, -1.484375, 1.455078125, -0.6357421875, 0.368896484375, -1.7080078125, -0.2181396484375, 1.2939453125, 0.447998046875, -0.2763671875, -0.74560546875, 0.225341796875, 0.307861328125, -0.42236328125, 1.33203125, -2.205078125, -0.17333984375, 1.4111328125, 0.325439453125, 0.6552734375, 0.191650390625, 0.355224609375, 0.21533203125, 0.332275390625, 0.92578125, -0.2666015625, -0.89794921875, -1.16796875, 0.5322265625, 0.56591796875, 0.6689453125, -0.6826171875, 0.301025390625, 1.6259765625, 0.5634765625, -1.546875, -0.62255859375, 0.75634765625, -0.623046875, 0.92822265625, -1.1865234375, -1.0234375, -1.08203125, -0.154296875, -0.810546875, -1.2470703125, 0.6943359375, 0.7060546875, 0.94921875, 0.6689453125, 1.08203125, -0.92724609375, 0.039642333984375, 0.71337890625, 1.05859375, 0.2135009765625, 1.841796875, -0.3994140625, -0.104736328125, -0.76806640625, 0.4990234375, 1.123046875, -0.07952880859375, 0.379638671875, -1.078125, -0.0165557861328125, -0.38427734375, 0.52392578125, -0.0877685546875, 1.412109375, -1.3681640625, 0.8623046875, -0.291259765625, -0.84716796875, -1.1396484375, 1.47265625, -1.3193359375, -0.46826171875, -1.46875, -0.29833984375, -0.37109375, -1.7783203125, 0.406005859375, 0.80615234375, -0.281494140625, 0.810546875, -0.290771484375, 0.88134765625, 0.263671875, -1.595703125, 0.09918212890625, -1.3125, -1.4267578125, 1.1279296875, -2.59765625, 0.732421875, 0.1612548828125, -0.372802734375, 1.068359375, -0.7294921875, 0.0014486312866210938, -0.2744140625, 0.830078125, 1.3134765625, -0.11041259765625, 0.35205078125, -0.52587890625, 0.266845703125, -0.2685546875, -0.3525390625, 1.337890625, 2.228515625, 2.357421875, 1.8994140625, 1.375, -1.75, 0.317138671875, 0.55859375, -0.60546875, -0.34716796875, 0.4013671875, 0.8642578125, -0.61474609375, 0.367919921875, 0.568359375, 0.4423828125, -1.0263671875, -1.923828125, 1.4462890625, 1.1005859375, 0.82470703125, -0.60205078125, -1.3486328125, -0.7880859375, 0.576171875, 0.253173828125, -0.2254638671875, -0.86083984375, -1.3486328125, -1.322265625, 0.415771484375, 0.301513671875, 0.09857177734375, 0.50146484375, 0.1444091796875, 1.6728515625, -0.029449462890625, -2.708984375, 0.1680908203125, -0.67578125, 0.56201171875, -0.1983642578125, 0.64306640625, -0.348876953125, -1.5263671875, 1.6083984375, 0.60888671875, 0.27001953125, 1.0263671875, 0.2890625, 1.2373046875, -1.7255859375, -1.0263671875, -0.64501953125, 1.6474609375, -1.2705078125, -0.67919921875, -0.1866455078125, 0.268798828125, 0.69970703125, 0.5703125, 1.552734375, 0.347412109375, -0.428955078125, 1.19921875, -0.2271728515625, 0.83056640625, 0.59423828125, -0.376708984375, -0.1417236328125, -1.0048828125, -0.328125, 0.57568359375, -0.5400390625, 0.4365234375, -0.23046875, -0.006145477294921875, -0.6982421875, 1.109375, 0.155029296875, -0.89306640625, -0.0192718505859375, 0.2474365234375, 1.3583984375, 0.33251953125, 0.91162109375, -1.5537109375, -1.9921875, -0.56640625, -1.3515625, 0.480224609375, 0.72265625, -0.243408203125, 1.1044921875, 1.041015625, 0.09637451171875, 0.34814453125, 0.1654052734375, -0.1217041015625, 0.63134765625, -2.3828125, -0.031951904296875, -0.436279296875, -1.3896484375, -0.2413330078125, 0.97265625, 2.646484375, 0.6455078125, 0.492431640625, -0.2017822265625, -0.521484375, 1.478515625, 1.4541015625, 1.5478515625, -0.4833984375, 1.0234375, 0.42578125, 0.76171875, -0.68701171875, 0.419189453125, 0.417236328125, -0.1494140625, -0.8212890625, 0.609375, 0.78369140625, 0.791015625, -0.2076416015625, -1.962890625, -0.61376953125, -1.5615234375, -1.212890625, 0.059661865234375, -0.3486328125, 1.8349609375, 0.0270538330078125, 0.219482421875, -0.62744140625, -0.9462890625, -1.1728515625, -1.205078125, 0.397216796875, -0.50048828125, -0.449462890625, -1.0439453125, -0.79638671875, 0.48583984375, -1.0986328125, 1.20703125, 0.83642578125, 0.78955078125, 0.48193359375, 1.05859375, -0.29248046875, -1.0537109375, 1.1025390625, 0.444091796875, 0.2408447265625, -0.1923828125, 0.81982421875, -0.127197265625, -0.0501708984375, 0.2181396484375, -0.159423828125, 0.73779296875, 0.0823974609375, 0.35400390625, 0.086181640625, 0.427978515625, 0.1551513671875, 0.43017578125, -1.306640625, -1.0615234375, 0.290771484375, 2.12890625, -1.572265625, -2.46875, 1.3037109375, 0.80029296875, -0.22021484375, 0.306884765625, -0.59423828125, 1.1318359375, 1.74609375, -0.231201171875, -0.98974609375, -1.2138671875, -0.304931640625, 0.86376953125, -0.59033203125, 0.07598876953125, -0.300537109375, 0.482666015625, -0.41162109375, -0.310791015625, 1.837890625, 0.140625, -0.07855224609375, 0.1849365234375, -0.37451171875, 1.828125, 1.5732421875, 0.48486328125, -0.408935546875, 1.7177734375, 0.31005859375, -0.2392578125, 0.6962890625, 0.50048828125, -0.7568359375, -0.64208984375, -0.12744140625, -0.397216796875, -0.7119140625, -0.81201171875, 0.142333984375, 0.98876953125, -0.27880859375, -0.29248046875, -0.68505859375, 0.1710205078125, 0.77783203125, 0.98583984375, -0.09051513671875, -0.1142578125, -0.344482421875, -0.64892578125, -2.70703125, 0.96533203125, 0.09490966796875, -0.1092529296875, -1.880859375, -0.62841796875, 0.70556640625, 1.6865234375, -1.0947265625, -0.82861328125, -0.2276611328125, 0.08624267578125, 0.7421875, 0.62841796875, 0.402099609375, 1.583984375, -0.25, 1.2529296875, -1.2900390625, 1.1904296875, -1.064453125, 1.744140625, 2.033203125, -0.333251953125, -0.71923828125, -1.603515625, 1.3720703125, 0.018157958984375, -0.43994140625, -1.013671875, 0.294921875, -2.072265625, -0.6943359375, -0.1116943359375, -2.369140625, -1.4111328125, 2.427734375, 0.55322265625, 2.9921875, 0.66259765625, 0.92724609375, -0.97119140625, 0.60302734375, -0.2587890625, 0.1251220703125, 0.5029296875, 1.1484375, 0.2379150390625, 1.173828125, 0.4248046875, 1.57421875, -1.7822265625, -0.7265625, 0.7353515625, -0.499755859375, 1.0302734375, 1.3330078125, -1.2509765625, 0.90234375, -1.19140625, -1.34765625, -0.91943359375, 0.5634765625, -0.85107421875, -0.5849609375, 0.419921875, 0.40673828125, 0.9404296875, -1.072265625, -0.89892578125, -0.75048828125, 1.3798828125, -0.48974609375, -0.53955078125, 0.04669189453125, -0.6708984375, -0.892578125, 0.89453125, -0.40625, -0.50732421875, 0.67724609375, -1.3701171875, -0.374267578125, -1.4375, -0.63525390625, -0.974609375, -0.1444091796875, 0.86279296875, 0.6826171875, 1.2802734375, -0.327392578125, 1.3193359375, -0.94873046875, -0.0660400390625, -1.880859375, 0.916015625, -1.123046875, 0.96337890625, -1.0625, -0.716796875, -0.69873046875, 1.1337890625, 0.32373046875, -0.94970703125, -0.8857421875, -0.472412109375, 0.60107421875, 1.5751953125, -1.11328125, -0.438232421875, -1.603515625, -0.76611328125, -0.75048828125, 0.8701171875, -0.216796875, 0.7109375, -1.2353515625, -0.83740234375, -0.84521484375, -0.91455078125, 0.337158203125, 1.4619140625, 1.9755859375, 0.451904296875, 1.0283203125, 0.89306640625, -0.309814453125, 0.4775390625, 1.59765625, 0.8095703125, 0.033660888671875, -0.253662109375, -0.1182861328125, 0.52294921875, -0.364501953125, -0.413330078125, -0.6728515625, -0.9599609375, -2.15625, -1.33984375, 0.66650390625, 0.284912109375, -3.39453125, -0.67724609375, 1.626953125, 0.2353515625, -0.36865234375, 0.55615234375, 1.025390625, 1.9228515625, 0.70263671875, 0.29443359375, -1.05078125, 1.6826171875, 0.6181640625, -0.04376220703125, 1.505859375, -1.3896484375, -0.319580078125, 2.03515625, -0.26318359375, -0.580078125, 0.5078125, -0.0024776458740234375, -1.1298828125, 0.2469482421875, 0.254150390625, 0.4267578125, 0.1822509765625, -1.0029296875, -0.75341796875, -0.045867919921875, -0.2098388671875, -0.83935546875, 1.228515625, 2.0859375, 0.199462890625, -1.423828125, -0.66455078125, -1.2890625, 0.177978515625, -1.087890625, -0.91748046875, -0.7470703125, -1.2705078125, 0.141845703125, 0.86181640625, -0.51513671875, -0.0029277801513671875, -0.332763671875, -0.69189453125, 0.001251220703125, -1.4716796875, 0.76513671875, 2.11328125, -1.1943359375, 1.384765625, 1.08984375, -1.3857421875, 0.428955078125, 0.73876953125, -0.26708984375, -0.1893310546875, 1.501953125, -0.6494140625, 1.3115234375, 2.9296875, -0.58154296875, 0.271484375, 0.51904296875, 1.2529296875, -1.46484375, 1.1259765625, 0.09197998046875, 1.58203125, -1.1162109375, 0.5869140625, -0.350830078125, 1.1328125, -0.354736328125, -0.0943603515625, 0.46630859375, -2.078125, 0.5498046875, -0.7451171875, 0.45361328125, -1.0078125, 1.6845703125, 0.2298583984375, 0.75, -2.31640625, 1.291015625, 0.95556640625, 0.4404296875, 0.382080078125, 0.52783203125, -1.18359375, 0.01259613037109375, 2.130859375, 0.07208251953125, 0.349365234375, -1.8720703125, -1.4296875, -0.135498046875, -0.28564453125, -0.10833740234375, -0.312744140625, -0.697265625, -0.5810546875, -1.0419921875, 0.0555419921875, -0.9306640625, -0.26708984375, 0.459716796875, -0.86767578125, 0.1829833984375, 0.88427734375, 0.4833984375, 1.5029296875, -0.73876953125, 1.0048828125, -0.1900634765625, 0.0010585784912109375, 2.376953125, 1.544921875, 1.564453125, -0.1312255859375, 0.31591796875, -0.2235107421875, -0.1302490234375, -0.0127716064453125, -0.06402587890625, -0.66650390625, -0.77734375, -0.157470703125, 0.0638427734375, -0.65966796875, -0.615234375, 0.4853515625, -1.2763671875, -2.623046875, -1.5078125, -1.126953125, 0.047698974609375, -0.1778564453125, 1.7421875, -0.14892578125, -0.1219482421875, 1.1630859375, -0.857421875, 0.47412109375, 0.46044921875, -0.96337890625, -0.01016998291015625, -1.46875, 0.8447265625, -0.0234832763671875, 0.042388916015625, -1.2158203125, 1.447265625, 0.492919921875, 1.1484375, -0.389404296875, -1.0185546875, 0.9619140625, -0.71875, 1.0419921875, -0.83837890625, -0.6923828125, -0.120849609375, -0.2236328125, -0.12939453125, 1.5068359375, 0.80224609375, -1.2451171875, 1.1923828125, -1.900390625, 1.5263671875, -0.02813720703125, 0.136962890625, 0.369873046875, -0.93505859375, 0.16650390625, 1.0126953125, 0.1513671875, -0.90185546875, 0.6650390625, 0.468994140625, -0.406982421875, 0.463134765625, -0.591796875, -0.79150390625, -0.12042236328125, 0.98046875, 1.93359375, 1.4052734375, 0.9375, -0.2252197265625, 1.158203125, -0.1348876953125, -0.0210723876953125, 2.1875, 0.78662109375, 1.1708984375, 1.3603515625, 0.79443359375, -0.080322265625, 1.2431640625, 0.771484375, 0.185791015625, -0.755859375, -0.33740234375, 1.7607421875, -1.0615234375, -0.50146484375, 0.1898193359375, 2.23828125, 0.1466064453125, 0.301025390625, -0.461181640625, 0.11669921875, 1.478515625, 1.0888671875, 1.650390625, -0.65869140625, 0.89111328125, 1.068359375, -0.2269287109375, -0.52197265625, 2.09765625, -0.7578125, -1.052734375, -0.9462890625, 0.2398681640625, -1.138671875, -1.9443359375, 0.529296875, 1.3203125, 0.97021484375, 1.0478515625, -1.2548828125, 0.6806640625, 0.15673828125, 1.5888671875, 0.092041015625, 1.673828125, -0.33203125, 1.9990234375, -1.8330078125, 0.83349609375, 0.83056640625, 0.027130126953125, -1.3173828125, -0.329833984375, -0.24951171875, 0.57373046875, -1.6796875, 1.2265625, 0.1719970703125, 1.2802734375, 1.1943359375, 1.6083984375, 0.277099609375, -0.814453125, -0.94091796875, 1.3984375, -2.287109375, -1.302734375, 0.8154296875, 0.48388671875, -1.263671875, -0.70458984375, -0.900390625, 1.2626953125, -0.66943359375, -0.081787109375, -0.2138671875, -0.599609375, -0.72314453125, -0.6181640625, -0.17724609375, 1.1005859375, -0.179931640625, 1.234375, -1.2998046875, 0.72509765625, 0.12115478515625, -1.5244140625, 0.31494140625, 2.888671875, 0.5654296875, -0.484619140625, 0.61572265625, 1.126953125, -0.0960693359375, 0.76806640625, 0.91455078125, 0.331298828125, -0.8310546875, -1.22265625, 0.59521484375, 0.046844482421875, 0.2587890625, -0.028045654296875, -0.64990234375, 0.8896484375, 1.24609375, 0.642578125, 0.1483154296875, 0.884765625, 2.28515625, 0.30224609375, 0.501953125, -0.68310546875, 0.3154296875, -1.064453125, -0.4375, -1.037109375, 1.6640625, 0.30615234375, -0.445068359375, 1.25, -0.5029296875, -1.17578125, -2.28125, 0.8349609375, -0.87890625, 0.333251953125, -1.4228515625, 0.724609375, 0.0694580078125, 1.685546875, -0.1492919921875, -0.344970703125, 2.03515625, 0.4375, -0.345703125, -0.41943359375, -0.3779296875, 0.2215576171875, 1.958984375, 1.0322265625, -0.99658203125, 0.763671875, 0.1595458984375, -1.6123046875, 0.8486328125, 2.021484375, -2.16796875, -0.191162109375, 1.5869140625, 0.358154296875, 0.4267578125, 2.525390625, -1.1181640625, 0.057861328125, -1.37890625, 0.62646484375, -1.1083984375, -0.182373046875, -0.04693603515625, -1.876953125, -2.111328125, 0.732421875, 1.3134765625, 2.509765625, 0.58984375, 0.2210693359375, 0.1676025390625, -1.189453125, 0.69287109375, 1.603515625, -1.3330078125, -0.791015625, -0.97607421875, 0.05035400390625, -0.56591796875, 0.30224609375, -0.0948486328125, -0.1490478515625, -1.03125, -0.1541748046875, -0.6376953125, 0.199951171875, 0.47509765625, 1.6806640625, -1.7646484375, -0.56591796875, 0.6435546875, 1.4365234375, -0.421142578125, -0.5986328125, -3.40625, 0.6552734375, 1.3310546875, 1.0849609375, 1.09765625, -0.221923828125, 0.073486328125, -0.0479736328125, 0.1802978515625, 0.467529296875, -0.79248046875, 0.7373046875, 1.740234375, -0.69677734375, 0.039947509765625, -0.357177734375, 0.322998046875, -1.263671875, 0.61669921875, 0.900390625, 0.63671875, 0.57763671875, -0.63525390625, -0.08514404296875, 1.0, 1.03515625, -0.26220703125, -2.86328125, -0.68359375, -0.06683349609375, 1.232421875, 0.303466796875, -0.07440185546875, -0.7080078125, 0.072265625, 0.50927734375, 0.306640625, 0.67724609375, -1.7802734375, -0.46728515625, 0.9609375, -0.689453125, 1.1796875, -0.271240234375, 0.35400390625, -0.779296875, -0.958984375, 1.08203125, 1.8447265625, 0.474853515625, -0.2568359375, 0.4189453125, -1.404296875, -0.0270233154296875, -0.193603515625, -1.1259765625, 0.416748046875, 0.1451416015625, -1.0498046875, -1.935546875, 1.6630859375, -1.064453125, -0.64501953125, 2.005859375, -0.90625, 0.51904296875, -0.57275390625, -1.1279296875, -0.498779296875, 0.544921875, -0.00939178466796875, 1.08203125, -1.4970703125, -0.6572265625, 0.456298828125, -0.482666015625, -1.41796875, 0.232421875, -1.6689453125, -0.69287109375, -0.317138671875, 0.89306640625, -0.2296142578125, 0.0165252685546875, -0.63623046875, 1.3369140625, -0.474853515625, -1.369140625, 0.1600341796875, -0.73779296875, 0.411865234375, -0.411865234375, 0.3095703125, -0.12890625, -1.1025390625, 1.0966796875, 1.841796875, -0.364990234375, 1.1201171875, 0.47265625, 0.97900390625, -0.280517578125, -0.53515625, -0.038360595703125, -0.379638671875, 0.75244140625, -0.276611328125, -0.55322265625, 0.486083984375, 0.69091796875, -0.2352294921875, 0.90478515625, -0.44921875, -0.587890625, -0.93115234375, -0.28564453125, -0.0117950439453125, -0.1431884765625, 0.82568359375, -0.78955078125, 0.53662109375, 0.6904296875, 1.1767578125, 0.67236328125, -0.8466796875, 0.63720703125, -1.6650390625, -0.283935546875, 0.47509765625, -0.43701171875, 1.333984375, 0.2431640625, -0.347412109375, 0.373046875, 0.07208251953125, -2.83203125, 0.939453125, -0.9033203125, 0.58056640625, 1.6748046875, -1.6884765625, -0.417724609375, -0.0288848876953125, -0.277099609375, 0.47607421875, 2.212890625, 2.26171875, 0.84521484375, 0.50439453125, -1.44921875, 0.09564208984375, -0.7568359375, -0.096923828125, -0.86572265625, 2.1640625, -0.5146484375, 0.472900390625, 0.281494140625, -0.8896484375, -0.367431640625, -0.48095703125, 0.271728515625, 0.2568359375, 0.16552734375, 0.6337890625, 0.24609375, 1.9521484375, 0.83984375, 0.60986328125, 0.2467041015625, -0.69189453125, -0.8486328125, 1.130859375, -2.537109375, -2.642578125, -1.17578125, 0.27099609375, 0.69775390625, 0.1640625, 1.298828125, 0.386962890625, 0.97802734375, -0.4912109375, -2.67578125, 1.0791015625, -0.43994140625, 1.2822265625, 0.921875, -0.86865234375, 0.026580810546875, 1.18359375, -0.79150390625, -1.599609375, -0.350830078125, 1.0009765625, 0.02325439453125, -1.306640625, 0.8583984375, 1.3017578125, -1.1171875, -0.70654296875, -1.2392578125, -0.7958984375, -0.62646484375, -0.03765869140625, -0.8466796875, 0.339111328125, 0.09735107421875, -0.33447265625, -0.5068359375, -1.982421875, 1.130859375, 2.24609375, -0.3486328125, 0.66748046875, 1.1787109375, 1.525390625, -0.03961181640625, 2.017578125, 1.0615234375, 1.7314453125, -0.87744140625, 0.75439453125, -1.263671875, 1.6328125, -0.85302734375, -0.4248046875, 0.03485107421875, 2.373046875, -0.37548828125, 0.611328125, -0.5234375, 0.1083984375, 1.3701171875, -1.29296875, -1.140625, 1.169921875, -1.3369140625, 0.93212890625, -0.5791015625, 1.6328125, 0.99609375, 0.646484375, 0.93701171875, 0.49755859375, 1.0888671875, -1.177734375, -0.3125, 2.861328125, -1.83203125, -0.465576171875, 0.30126953125, 0.0758056640625, -0.025604248046875, 0.544921875, 0.849609375, 0.325927734375, 0.0638427734375, 0.65869140625, 1.0205078125, 1.4091796875, -1.716796875, -0.95068359375, -0.71240234375, -0.66064453125, 0.495361328125, 0.89794921875, -0.771484375, -1.3828125, -0.216796875, -0.307373046875, 0.96240234375, 1.1162109375, 0.207763671875, 1.4384765625, 0.50537109375, -0.025970458984375, 0.970703125, 0.25537109375, -1.5478515625, -0.058197021484375, 0.814453125, 0.736328125, 0.63720703125, 0.57763671875, -0.09613037109375, -0.9970703125, -2.80078125, -1.982421875, 0.3466796875, -0.10650634765625, -0.39501953125, 1.724609375, 1.681640625, -0.302978515625, -0.7529296875, 0.771484375, -0.046630859375, -0.87744140625, 1.1005859375, -1.3681640625, -1.14453125, 0.7109375, -0.24951171875, 0.2452392578125, -1.4541015625, 0.71728515625, 0.74560546875, -2.197265625, 0.36376953125, -2.330078125, 0.6376953125, 1.5322265625, -0.57373046875, 1.072265625, -0.71435546875, 1.837890625, 0.626953125, 0.0911865234375, 0.84521484375, -3.009765625, -1.4091796875, -0.82958984375, 0.85546875, 0.283447265625, -1.4443359375, 0.82177734375, 1.4287109375, -1.181640625, -1.1826171875, 1.1396484375, -0.32373046875, -0.04937744140625, -1.154296875, -0.7958984375, -0.86474609375, -0.625, -0.45947265625, 1.4296875, 0.15869140625, 0.160888671875, -0.6455078125, -2.8125, 0.294921875, 1.5634765625, 0.62744140625, -0.92041015625, 0.3916015625, 1.2216796875, -0.6396484375, -0.2364501953125, -0.77734375, 1.232421875, -0.34423828125, -0.208740234375, -0.5478515625, -0.400390625, -0.7158203125, -0.20654296875, -0.349365234375, -0.91845703125, -0.322021484375, 0.11883544921875, 2.228515625, -0.5859375, 0.35205078125, -1.2685546875, 0.65673828125, -0.5283203125, 0.5712890625, -0.4443359375, -0.56201171875, 0.06695556640625, -0.68310546875, 0.12493896484375, -0.6044921875, 0.1114501953125, -1.470703125, -0.336669921875, 0.455810546875, -0.31640625, -0.436279296875, -0.1917724609375, 0.60107421875, -1.4921875, 0.5595703125, -0.232177734375, 0.62353515625, 0.765625, 0.020263671875, -0.94677734375, -0.284423828125, 0.57763671875, -1.623046875, 0.434326171875, -0.170166015625, -0.2425537109375, -1.154296875, 0.281005859375, 0.72412109375, -0.4501953125, 0.041107177734375, 0.76025390625, -1.0771484375, 0.101318359375, 1.34765625, 0.09527587890625, 0.1595458984375, -0.95166015625, 0.233642578125, 0.296875, -0.3125, -0.94921875, -2.283203125, 0.46044921875, 0.48974609375, 1.01171875, 0.079345703125, -1.5009765625, -0.093994140625, 0.9736328125, 0.405029296875, 1.958984375, -0.705078125, 0.3779296875, 0.0169219970703125, -1.2587890625, 0.8740234375, 0.859375, 0.03167724609375, 0.359619140625, -0.06982421875, 0.206298828125, -0.61572265625, 1.0185546875, -0.66455078125, -0.33447265625, -0.1513671875, 1.05078125, -0.74267578125, 0.02130126953125, 0.501953125, 0.923828125, 0.24267578125, -0.50244140625, -0.450927734375, 0.4140625, -0.84228515625, -1.423828125, -1.2138671875, -0.41845703125, -0.95849609375, 0.53759765625, -0.85791015625, 1.99609375, 1.02734375, 0.072509765625, 1.4453125, 0.438720703125, 0.6240234375, 0.1470947265625, 0.057891845703125, 0.52001953125, 0.544921875, 0.8291015625, -0.90234375, -0.433837890625, 0.30908203125, 0.0736083984375, 0.9404296875, -0.30859375, -1.0341796875, 0.8544921875, 1.4072265625, -0.37890625, -0.63330078125, 0.30029296875, 2.40234375, -0.55029296875, -0.8154296875, 0.4619140625, -1.1123046875, 0.5224609375, -1.1953125, 0.69677734375, -0.271240234375, 0.5595703125, -0.03570556640625, 0.276611328125, -0.896484375, 0.297607421875, -0.27490234375, 0.391845703125, 0.283935546875, 0.1329345703125, -0.57568359375, 0.171875, -0.3017578125, -2.203125, -0.2509765625, 0.17333984375, 0.310546875, -1.7607421875, -1.9404296875, 0.16259765625, -0.393798828125, -0.360595703125, -1.3916015625, 0.6064453125, -0.24267578125, -1.0703125, -1.0185546875, -0.08245849609375, -0.355224609375, -1.578125, -0.1829833984375, -0.37890625, -1.6826171875, -0.492919921875, 1.1982421875, 2.2109375, 0.24853515625, 1.5830078125, -1.6162109375, -1.421875, -0.5615234375, 1.48828125, -0.7646484375, 0.26611328125, -0.436279296875, 0.356201171875, 0.81884765625, 0.8427734375, -1.1962890625, -0.47509765625, -1.578125, -1.1904296875, 0.2056884765625, 1.455078125, 1.818359375, 0.61669921875, -1.521484375, -0.97607421875, -1.0048828125, 1.3046875, -0.4599609375, 1.1220703125, 0.75537109375, 0.09637451171875, 2.044921875, -0.423095703125, -0.47021484375, 0.1376953125, -0.69775390625, -0.21875, 0.382568359375, -0.412353515625, -0.203369140625, -0.446044921875, 0.9482421875, 0.55517578125, -1.283203125, -0.88232421875, 1.919921875, 0.56201171875, 1.0986328125, -1.048828125, -0.298095703125, -1.0732421875, 0.3427734375, 1.08203125, -1.2109375, -0.07598876953125, -0.2412109375, 0.1337890625, 1.64453125, 0.201171875, -0.1549072265625, -0.3203125, 0.282958984375, -0.1319580078125, 0.3486328125, 1.0703125, 0.80322265625, 1.59765625, -0.83935546875, 0.81884765625, -1.5205078125, 0.03277587890625, 0.06842041015625, -2.255859375, 1.09765625, -0.92236328125, -0.84423828125, -0.51416015625, -2.056640625, -0.23193359375, -0.50830078125, 1.328125, -1.693359375, -1.1025390625, 1.0029296875, -0.1412353515625, -0.57080078125, -0.09600830078125, -1.578125, 0.53857421875, 1.607421875, -0.74169921875, 0.8701171875, 0.0225982666015625, 0.483154296875, -0.958984375, -1.1337890625, -1.01953125, -1.28125, 1.9521484375, 1.080078125, 1.404296875, -0.580078125, -1.1796875, -0.254638671875, 1.1259765625, -0.1181640625, 0.9169921875, -0.9052734375, 1.4150390625, -0.006725311279296875, -0.9638671875, -0.1304931640625, 0.8203125, 0.6875, 0.314697265625, -0.404052734375, -1.1318359375, 0.91552734375, -1.2705078125, 0.9091796875, 0.077880859375, -0.0290985107421875, 0.13037109375, 0.80322265625, 0.04559326171875, 0.02545166015625, 1.6337890625, -0.164794921875, 0.953125, 1.365234375, -1.7890625, -0.59130859375, -0.2374267578125, 0.6103515625, -0.052764892578125, -0.80712890625, 0.221435546875, -0.050445556640625, 0.376953125, 0.492431640625, 1.3125, -0.10662841796875, -0.9521484375, 0.1632080078125, 0.05450439453125, -0.80029296875, 2.560546875, -1.43359375, 0.578125, 1.61328125, 0.1868896484375, -0.5888671875, 0.70556640625, 0.09454345703125, -0.5517578125, -0.80615234375, -1.4580078125, -1.744140625, -0.0109405517578125, 0.982421875, -0.80126953125, 3.458984375, 0.052947998046875, -0.200927734375, -0.333984375, 1.37109375, -0.998046875, -0.371826171875, 0.81396484375, -1.3876953125, -0.99072265625, -0.496337890625, -2.056640625, -0.1971435546875, 0.66259765625, -0.1932373046875, -0.25146484375, 0.1165771484375, 0.1502685546875, -1.0322265625, 0.7158203125, 1.048828125, 0.06640625, -0.187255859375, 0.04443359375, 0.01117706298828125, 0.51171875, -0.54296875, -1.1953125, -0.4375, -0.4599609375, 0.85400390625, -0.011993408203125, 0.297119140625, -1.6611328125, -1.017578125, 0.61962890625, 0.2568359375, 0.240234375, -0.8740234375, 0.56591796875, 1.7666015625, 0.96728515625, 0.71337890625, -1.09375, 0.307861328125, -0.442626953125, 0.556640625, -1.6357421875, -0.25732421875, -0.333740234375, 0.2318115234375, -0.0855712890625, 0.36376953125, -0.1759033203125, -0.880859375, 1.4326171875, 0.345947265625, -0.09967041015625, -0.222412109375, 1.337890625, 0.48974609375, 0.84814453125, -1.330078125, -0.64794921875, -1.3681640625, -0.30859375, -0.04058837890625, 0.2177734375, 0.36962890625, -1.9501953125, 1.9482421875, 0.343505859375, 0.1767578125, 0.63525390625, -1.044921875, 1.1376953125, -2.001953125, -0.0179290771484375, 0.288330078125, 0.99853515625, 0.23486328125, 1.2353515625, -2.201171875, 0.2113037109375, 0.366943359375, 0.794921875, 0.66552734375, 0.1224365234375, -0.0560302734375, 0.29052734375, 1.494140625, 1.0732421875, -0.765625, -0.65966796875, 0.60791015625, 2.509765625, -0.568359375, -1.3017578125, 1.6669921875, 0.5595703125, 0.58935546875, 1.3984375, -0.173583984375, -1.380859375, 0.9169921875, 0.1251220703125, -0.56884765625, -1.814453125, -0.572265625, 1.4658203125, 0.75244140625, -0.1444091796875, 0.8642578125, 0.87646484375, -0.70263671875, 0.63671875, -0.78076171875, -0.31640625, -1.7529296875, -0.3720703125, -0.0477294921875, 1.125, 2.462890625, 0.7763671875, 0.284912109375, -0.859375, -0.609375, -0.18359375, 1.2021484375, 1.0556640625, 0.39453125, 0.33349609375, -0.130859375, -0.8154296875, 0.44677734375, -0.5283203125, 1.2412109375, 1.3349609375, 0.46142578125, -1.8671875, 0.354736328125, 0.80517578125, 0.6533203125, 0.269775390625, 0.341552734375, 0.71142578125, 1.087890625, -2.064453125, 0.434814453125, -1.3154296875, -0.1990966796875, 0.1971435546875, 2.0234375, 1.669921875, -0.86669921875, 1.0576171875, -0.57080078125, 0.66845703125, -1.216796875, -1.0810546875, -0.310791015625, 0.263916015625, -0.83837890625, -1.255859375, -0.250244140625, -0.41650390625, -0.4638671875, -0.65478515625, 0.62109375, 0.16748046875, -1.0341796875, -0.2137451171875, 0.11956787109375, 0.64306640625, -1.611328125, -1.1455078125, -1.0322265625, -0.76025390625, 0.7919921875, -3.2890625, 0.460205078125, -0.264404296875, 1.33203125, -0.08984375, -0.33740234375, -0.24169921875, -0.2200927734375, 1.279296875, -0.5810546875, 0.7216796875, -0.798828125, -0.1873779296875, 0.0191497802734375, -1.2919921875, 0.27734375, -1.0634765625, -0.96533203125, -1.7958984375, 0.93505859375, -0.35400390625, 0.09075927734375, -1.8251953125, 1.0556640625, -0.223388671875, -0.625, 0.50732421875, -0.339111328125, -0.134521484375, -0.52001953125, 0.51318359375, 0.77783203125, 1.7626953125, -0.2132568359375, 1.005859375, 0.673828125, 0.88134765625, -1.9609375, -0.30712890625, -1.494140625, -0.0738525390625, 0.69921875, -0.2744140625, -0.30908203125, 0.42919921875, 0.329833984375, 1.4189453125, -0.6875, 2.67578125, -0.55322265625, 2.048828125, -0.053436279296875, -0.489990234375, -0.430908203125, -0.1678466796875, 1.390625, -0.2098388671875, -0.0997314453125, -0.476318359375, 0.436279296875, 1.1806640625, 1.048828125, 1.12109375, -0.1163330078125, 0.6015625, 0.80419921875, -0.380859375, -2.470703125, 0.359130859375, 0.396240234375, -1.494140625, -1.0517578125, 0.169921875, 0.250732421875, -0.402587890625, -1.884765625, 1.255859375, 0.1580810546875, -0.038970947265625, -1.6962890625, 0.765625, 0.59521484375, -0.59326171875, -0.9130859375, -0.0771484375, -0.658203125, -0.41357421875, -0.31298828125, -2.455078125, 1.359375, -1.4404296875, 1.236328125, -0.361328125, -0.19873046875, 0.2158203125, -0.93310546875, 0.327880859375, 0.1982421875, 0.471923828125, 1.8203125, -0.74267578125, -0.0777587890625, -0.09832763671875, 0.2213134765625, 1.3759765625, -0.54345703125, 2.001953125, -0.1561279296875, -1.28125, -0.483642578125, -0.416015625, -0.30419921875, 0.145263671875, -1.8017578125, 0.58544921875, 1.1669921875, 1.2216796875, -0.66552734375, 0.3681640625, 1.443359375, -0.56640625, 0.48486328125, -1.89453125, -0.77880859375, -0.5771484375, -0.5458984375, -1.6953125, 0.3193359375, 1.3525390625, 1.833984375, -1.28515625, -0.42138671875, 0.529296875, 0.623046875, 0.1719970703125, 0.51708984375, 0.77099609375, -1.28515625, 0.60693359375, -0.476318359375, -0.5576171875, -0.292724609375, 0.302001953125, 0.72265625, -1.0341796875, -1.580078125, -0.779296875, -1.830078125, 0.52978515625, -0.1307373046875, -0.9052734375, -1.525390625, -1.2607421875, 1.166015625, -0.320556640625, 4.109375, -1.0419921875, 0.755859375, 0.642578125, 2.2890625, 1.326171875, 0.2276611328125, 1.23828125, -0.2646484375, -0.06768798828125, -0.206787109375, -0.45556640625, 0.4248046875, 0.188232421875, -0.0965576171875, -0.7197265625, 1.34375, -0.748046875, 0.98876953125, 0.4111328125, -0.51708984375, 0.438720703125, -0.90234375, -0.72509765625, -1.0400390625, 0.744140625, 1.05859375, 0.88623046875, -0.50537109375, -0.623046875, -0.1435546875, 0.345458984375, -0.876953125, -0.88916015625, 1.0234375, -0.79638671875, 0.414306640625, 1.201171875, -0.310791015625, 0.87060546875, -0.493408203125, -0.06866455078125, -0.623046875, 0.68701171875, -0.4248046875, -0.0596923828125, -0.358154296875, 2.31640625, 0.163818359375, 0.90625, -1.373046875, -2.013671875, -0.099609375, -0.74560546875, 1.6142578125, 0.88427734375, -0.1558837890625, 0.397216796875, 1.9482421875, 0.308349609375, -1.205078125, 1.271484375, -0.5703125, -0.83984375, 2.48828125, -0.630859375, 0.1998291015625, 0.72265625, 0.1080322265625, -0.2166748046875, 2.58203125, 0.77685546875, -1.4404296875, -0.0289764404296875, 0.65771484375, -0.4326171875, -0.031890869140625, -0.61669921875, -0.06695556640625, -0.759765625, 2.9375, 0.382080078125, -1.765625, -0.5068359375, 0.0277557373046875, 0.859375, -0.9169921875, -0.64697265625, -1.00390625, 1.1884765625, 0.67626953125, 0.3623046875, -0.284423828125, 0.1129150390625, 2.3125, 1.7685546875, -0.0199432373046875, 1.66015625, -1.689453125, 0.33447265625, -1.697265625, 1.123046875, -0.55859375, 1.49609375, -0.0164947509765625, 0.41943359375, 0.1947021484375, -0.98193359375, -0.5947265625, 0.1097412109375, -0.2724609375, 0.4638671875, -1.5048828125, -0.15625, 0.67578125, -1.6669921875, -0.35400390625, 0.6162109375, -0.74267578125, -0.72607421875, -0.9453125, -0.6220703125, -0.01152801513671875, 0.162353515625, 0.4931640625, 0.64404296875, 1.4677734375, -0.2578125, 0.65966796875, -0.67724609375, 0.3818359375, -0.1798095703125, -0.07366943359375, -0.78857421875, 0.361328125, -0.375244140625, 0.080810546875, -2.236328125, 1.40625, -0.14306640625, 0.491455078125, 0.3056640625, -0.69775390625, 0.08258056640625, 1.970703125, -0.277099609375, -2.2109375, 1.146484375, -1.9150390625, -0.083251953125, -0.7939453125, 0.005519866943359375, 0.29638671875, 0.59423828125, -0.56103515625, -1.7578125, -0.03668212890625, 0.20849609375, -0.48291015625, 0.5224609375, 1.3720703125, -1.8037109375, -1.576171875, -1.1669921875, -2.201171875, 0.183837890625, -0.72021484375, -1.5205078125, 0.30322265625, 0.90869140625, -0.054412841796875, 0.953125, 0.99609375, 0.27685546875, 0.31201171875, -0.7099609375, -0.68212890625, 0.6748046875, 0.345458984375, -0.51416015625, -0.27392578125, -0.64599609375, -0.0567626953125, -0.44970703125, 1.095703125, -0.5498046875, -0.08880615234375, -0.93505859375, 0.7177734375, -0.75390625, -0.2191162109375, 0.0650634765625, 0.258056640625, 1.3017578125, -1.892578125, 0.11846923828125, -1.1015625, 0.69775390625, -0.57080078125, 2.0546875, -1.26171875, -1.736328125, -0.054595947265625, -1.7109375, 0.9951171875, 0.470947265625, 0.360595703125, -0.357177734375, 1.873046875, 0.2900390625, 0.33935546875, -0.7919921875, 0.0019044876098632812, -1.1611328125, 0.9013671875, -0.139892578125, 1.9775390625, 0.94775390625, 0.3505859375, -1.265625, -0.379150390625, 0.329345703125, 0.00738525390625, -0.315185546875, -2.0078125, 0.32275390625, 0.087890625, -2.1171875, 0.0384521484375, 0.0294189453125, -0.94775390625, -0.423095703125, -0.2452392578125, 1.193359375, 0.414306640625, -1.017578125, -1.2216796875, -0.37744140625, 0.98779296875, 1.302734375, -1.0859375, 1.8955078125, 0.8408203125, -1.75390625, 1.6083984375, -0.337158203125, 0.126220703125, -0.95166015625, -0.50732421875, 2.03125, 0.1932373046875, -0.361083984375, -0.171630859375, 0.623046875, -0.9404296875, -0.043243408203125, 0.20166015625, -1.6767578125, 0.1943359375, -1.4912109375, 1.4775390625, -1.953125, 0.385498046875, -1.2177734375, -2.3515625, 0.25537109375, 0.67626953125, 1.712890625, 0.6083984375, 0.16650390625, -0.264404296875, -0.2210693359375, -1.8876953125, 0.133056640625, -0.0150146484375, 1.6787109375, 0.224609375, 0.035552978515625, 0.76416015625, 0.90234375, -0.7158203125, -1.033203125, -1.708984375, 0.289306640625, 1.2001953125, 1.0185546875, 1.11328125, 0.71630859375, 1.25, 2.1328125, 0.2177734375, 0.2344970703125, 0.4248046875, 0.9150390625, 0.56494140625, 0.06689453125, -1.197265625, 1.2744140625, -0.560546875, -2.16796875, 0.1429443359375, -0.215576171875, -0.61767578125, 1.7607421875, 0.66650390625, -1.2216796875, -2.126953125, 0.5234375, -0.96240234375, 0.318115234375, -1.1796875, 1.291015625, -0.82861328125, 0.189453125, 0.3134765625, 0.50537109375, 0.28515625, 0.12359619140625, 0.0887451171875, 3.162109375, 1.16796875, -0.96240234375, 1.013671875, 0.0135650634765625, -0.57666015625, 2.30859375, 0.268798828125, -0.78857421875, 0.2381591796875, -1.4453125, -0.287353515625, -0.07562255859375, 0.76806640625, -0.6923828125, 0.60595703125, -0.38037109375, 1.6142578125, -0.7275390625, 0.44873046875, -0.451416015625, -0.004978179931640625, -0.87841796875, -1.0498046875, -1.36328125, 0.75537109375, 0.3818359375, -1.5361328125, -1.6416015625, -0.96044921875, 0.413330078125, -1.66796875, 0.546875, 0.16552734375, 0.580078125, -1.171875, -0.93603515625, -1.1064453125, 0.2225341796875, -0.30029296875, -1.84765625, -0.6806640625, 1.09375, -0.451171875, 0.92724609375, 0.39794921875, -0.630859375, -1.06640625, 1.45703125, -1.5263671875, 0.533203125, 1.68359375, 1.8076171875, -0.611328125, 0.299072265625, 0.0858154296875, -1.529296875, -0.99072265625, -0.98681640625, 0.74658203125, -0.058197021484375, -1.4130859375, 0.837890625, -0.62548828125, -0.8642578125, -0.42919921875, -0.92578125, 1.1201171875, 1.125, -0.354248046875, 1.31640625, -0.086669921875, 0.7822265625, -2.197265625, 1.72265625, -0.473876953125, 0.25390625, -0.18994140625, -0.7509765625, 0.1661376953125, -1.47265625, 0.609375, -0.320556640625, -1.501953125, 0.416259765625, -1.1083984375, 0.5849609375, -1.482421875, 0.5732421875, 0.339599609375, 1.1474609375, -0.2169189453125, -0.08612060546875, 0.8662109375, -2.029296875, 0.79052734375, 1.048828125, 0.78759765625, -0.413330078125, -0.5263671875, -0.798828125, -0.30126953125, 0.8564453125, 1.82421875, 0.4267578125, -1.3916015625, -1.0380859375, -0.1553955078125, 0.0411376953125, -0.787109375, 0.0272064208984375, -0.5126953125, -1.4365234375, 0.376220703125, -0.82373046875, 0.1334228515625, 0.366455078125, 1.630859375, -0.326171875, -0.1207275390625, 1.021484375, 0.83740234375, -1.7587890625, 0.35986328125, -1.2734375, 0.2010498046875, -2.517578125, -1.9697265625, 2.396484375, -0.42578125, -0.4833984375, -0.0648193359375, 0.63671875, 0.73291015625, -0.07574462890625, 1.2216796875, 1.3984375, 1.12109375, -0.296142578125, -1.14453125, -0.279052734375, 0.6103515625, -0.470458984375, 0.9931640625, 2.052734375, 1.6162109375, -1.1328125, -2.00390625, 0.8759765625, -1.2685546875, -2.09765625, 0.437744140625, 0.76123046875, -0.0633544921875, 0.1075439453125, 0.31396484375, -0.2705078125, -0.06097412109375, -0.640625, 0.08984375, 0.290283203125, 1.7978515625, -0.372802734375, -1.0595703125, 0.1312255859375, 0.9580078125, -0.79248046875, 1.3779296875, -0.82470703125, -1.625, 1.306640625, -0.8193359375, 0.388427734375, -1.2392578125, 0.8115234375, -0.312744140625, -1.810546875, -0.50732421875, 2.271484375, 0.59375, -0.88671875, -0.4736328125, 0.473388671875, 0.74072265625, 1.1884765625, -1.51171875, -1.3271484375, -0.99853515625, 0.0010919570922851562, -0.89892578125, 0.734375, -0.50732421875, 0.1456298828125, 0.457275390625, 0.73828125, -1.0537109375, -1.671875, -0.97802734375, 0.66552734375, -1.173828125, 1.1181640625, -1.4462890625, -0.66064453125, -0.092041015625, 1.0986328125, 1.611328125, -0.4775390625, 0.681640625, -0.59423828125, 0.5615234375, 1.150390625, 0.763671875, -1.4521484375, 0.4443359375, 0.5185546875, -1.0966796875, 1.2275390625, -0.890625, -0.75830078125, -0.58203125, -0.020263671875, -0.299560546875, 0.40625, 0.552734375, -1.3212890625, -0.495361328125, -0.85498046875, 0.52685546875, -1.3583984375, 0.1180419921875, 0.587890625, 0.318115234375, 0.482421875, -0.53173828125, 0.7626953125, -0.19921875, 1.0654296875, 1.8466796875, 1.623046875, -2.69140625, -0.303466796875, -1.55078125, -0.184326171875, 0.66552734375, -1.484375, 0.2171630859375, 0.1767578125, 0.1798095703125, -0.45947265625, 0.3955078125, 0.53955078125, 1.05859375, -0.68017578125, -0.056671142578125, 1.146484375, 2.25390625, -2.162109375, 0.4140625, -0.58154296875, 1.416015625, 0.11285400390625, -1.1044921875, 0.8603515625, 0.352294921875, -0.318115234375, -1.466796875, -0.2000732421875, -0.337158203125, 0.7373046875, 0.62109375, 0.44140625, -1.1787109375, -0.3876953125, 0.345458984375, -0.82373046875, 2.681640625, 0.6240234375, -0.130126953125, -0.82568359375, 0.62158203125, 0.72021484375, -0.9384765625, -1.1142578125, 2.14453125, 1.8720703125, 1.330078125, -0.8056640625, -1.2490234375, -0.161376953125, -1.298828125, -0.12030029296875, -0.3671875, -1.62890625, -1.1787109375, 0.82421875, 1.697265625, -0.82421875, 1.1083984375, 0.53173828125, 1.123046875, 0.947265625, -0.58349609375, -0.039581298828125, 0.407958984375, 0.3818359375, 0.59423828125, -0.642578125, -0.280029296875, -0.173828125, -2.220703125, 0.58203125, 0.328857421875, 0.3505859375, 0.87744140625, 1.0224609375, -0.67236328125, 2.203125, 0.490234375, 1.2822265625, -0.67578125, 1.4912109375, -1.6865234375, 0.06280517578125, -0.57421875, -0.94873046875, 0.6435546875, -0.464599609375, -2.546875, 0.55517578125, -0.031707763671875, 1.126953125, -0.35791015625, -0.58935546875, -0.5888671875, -0.8095703125, -0.70751953125, 0.1453857421875, 1.306640625, 0.7080078125, 0.6689453125, -0.29736328125, -3.416015625, 0.320556640625, -1.0068359375, 0.00791168212890625, -0.418701171875, 1.119140625, 1.0634765625, 0.294189453125, -0.7099609375, -1.23046875, -0.1712646484375, -0.107177734375, 0.7421875, -1.560546875, -0.053009033203125, -1.396484375, -0.9990234375, 0.75390625, 1.09375, -1.19140625, 0.61474609375, -0.88232421875, 1.63671875, 0.21435546875, 2.509765625, -0.35009765625, 3.083984375, -0.98291015625, -0.1734619140625, 1.1865234375, -0.8193359375, 0.053253173828125, 0.59521484375, -0.09222412109375, -0.0693359375, 1.2763671875, -0.703125, 0.494140625, 0.71240234375, 0.174560546875, 0.35498046875, -0.634765625, 1.2607421875, 2.646484375, 0.288330078125, -0.7353515625, -0.316162109375, -0.91357421875, 0.1456298828125, 0.491455078125, -0.52734375, 0.85888671875, 0.243896484375, 1.439453125, -0.0201873779296875, -2.955078125, -0.0170135498046875, 0.32421875, 1.826171875, -1.205078125, -1.1474609375, -0.385009765625, 0.974609375, 2.203125, -1.248046875, 0.07940673828125, 0.259033203125, 0.94189453125, -0.5224609375, 1.4736328125, 0.52685546875, 0.80615234375, -0.68017578125, 1.263671875, 0.609375, -1.8359375, 0.56787109375, -2.22265625, -1.240234375, 0.1978759765625, 0.765625, -0.650390625, -0.029296875, 0.7373046875, 1.1669921875, 0.138916015625, 0.0196685791015625, -2.14453125, 0.48486328125, -1.5859375, -0.2044677734375, 0.55859375, -0.1851806640625, -0.72509765625, 1.0234375, -0.6982421875, -0.31982421875, -0.78076171875, 1.203125, 0.4248046875, 0.77001953125, 1.4189453125, 1.29296875, -0.9521484375, 0.45849609375, 0.470458984375, 2.701171875, -0.0830078125, -1.0595703125, -1.58984375, 0.451171875, -0.9921875, 0.79248046875, 0.57177734375, 0.11761474609375, -0.380615234375, -0.389404296875, -0.99755859375, -1.173828125, -1.6650390625, -0.059814453125, 0.0318603515625, -0.1806640625, -0.333251953125, 0.96337890625, 3.04296875, -1.037109375, 0.2437744140625, 0.437255859375, -0.56689453125, 0.1337890625, -1.8125, -1.7275390625, 0.07916259765625, -0.15869140625, -0.151611328125, 1.7109375, 0.492431640625, 0.78759765625, -0.22119140625, 0.9990234375, 0.51513671875, -0.55810546875, 0.65966796875, 1.2607421875, 0.142578125, -0.05999755859375, 0.93994140625, 0.438232421875, 1.0380859375, 1.1904296875, -0.03253173828125, -0.89501953125, -1.2216796875, -0.454833984375, 0.00658416748046875, -0.87548828125, 0.4150390625, 0.88330078125, 1.017578125, -1.16796875, -0.28369140625, 0.311767578125, -0.5712890625, 0.234375, -0.51708984375, 0.66845703125, 0.75634765625, 0.482666015625, 1.673828125, 1.525390625, 6.55055046081543e-05, -0.5615234375, -0.1871337890625, -0.0771484375, -0.77490234375, 0.60107421875, -1.2626953125, -0.2115478515625, -0.70263671875, -1.275390625, -0.1737060546875, -0.67578125, 0.60791015625, -0.59375, -0.77978515625, 0.303466796875, -1.06640625, -1.1640625, -0.9873046875, -0.29736328125, -0.3798828125, 0.7353515625, -1.1181640625, -0.2196044921875, 0.63623046875, 0.5869140625, -0.8115234375, -1.11328125, 0.83984375, 0.2286376953125, 0.927734375, 0.70849609375, 0.1373291015625, -0.308837890625, -1.27734375, 0.436767578125, -0.4150390625, 1.1259765625, 1.443359375, -0.033172607421875, -0.7353515625, 0.732421875, 0.55224609375, -0.399169921875, 0.60009765625, -0.387939453125, -1.111328125, -0.32958984375, 0.162109375, -0.9423828125, -0.430908203125, -0.68701171875, 0.85009765625, -1.1982421875, 0.05908203125, -0.39013671875, -1.125, 0.059906005859375, -2.1015625, 0.384765625, 0.07244873046875, -1.994140625, 2.02734375, 0.37939453125, -2.3671875, -0.07513427734375, 0.57958984375, -1.2236328125, 1.80078125, -2.064453125, 0.409423828125, -1.1787109375, 0.44091796875, 0.82177734375, -0.53125, 0.57763671875, 0.8291015625, -1.236328125, 0.5751953125, -0.48193359375, 1.396484375, 0.144775390625, 0.828125, -0.8349609375, 0.36865234375, 1.7265625, -0.41845703125, -0.81494140625, -0.6748046875, 0.8994140625, 0.3173828125, -0.295654296875, 0.4521484375, -0.164306640625, 0.56298828125, 0.3291015625, -1.103515625, 0.81494140625, 0.458984375, -1.0126953125, 1.0078125, 0.022125244140625, 2.626953125, -0.021881103515625, -0.318603515625, -1.2822265625, -1.60546875, -0.6572265625, -0.3115234375, 1.8916015625, -0.026153564453125, -1.7353515625, 2.126953125, -0.9658203125, -0.321044921875, 0.8359375, 1.3037109375, -0.52783203125, 0.062469482421875, -1.212890625, 0.677734375, -0.8505859375, 0.98291015625, -0.88134765625, -0.426025390625, 1.701171875, -0.2509765625, -0.187255859375, 0.2318115234375, 0.7265625, -0.166748046875, 0.05987548828125, 0.67236328125, -0.49853515625, -0.17431640625, -0.29931640625, 0.409423828125, 0.5419921875, -0.11346435546875, 2.677734375, -0.2548828125, 0.462158203125, 0.41015625, 1.283203125, -1.0390625, 0.322509765625, 1.9765625, -0.328857421875, -1.1142578125, 0.395751953125, 0.31982421875, -1.2880859375, 1.4228515625, 0.541015625, -0.145263671875, -1.3583984375, 0.79345703125, -0.9921875, -0.166748046875, -0.7119140625, -0.892578125, 1.7998046875, -0.40673828125, 1.3798828125, -0.615234375, 0.495849609375, -0.93359375, -0.09759521484375, 1.0107421875, 0.21484375, 1.1962890625, -1.421875, 0.765625, -1.748046875, -0.697265625, -1.8955078125, -1.5068359375, 1.5205078125, 0.05609130859375, 0.363037109375, 0.935546875, 0.66552734375, 0.2161865234375, 0.70068359375, 0.90771484375, 0.140380859375, 1.408203125, 0.509765625, -0.20458984375, -0.8544921875, -0.99755859375, 0.283447265625, -0.0567626953125, -1.1201171875, 1.8642578125, -2.23828125, -0.008087158203125, 0.28662109375, -1.2431640625, -0.68505859375, 0.025177001953125, -0.498779296875, 1.0751953125, 0.55908203125, -0.4921875, -0.654296875, -0.492919921875, -0.85009765625, 0.25244140625, -0.474609375, -1.109375, -1.9345703125, -0.533203125, 0.25244140625, -1.220703125, -1.1640625, 1.466796875, -0.496826171875, -0.368408203125, 0.486328125, 2.017578125, 0.58056640625, -0.3583984375, 0.61279296875, -0.003551483154296875, 0.355224609375, -0.61572265625, -0.7158203125, -0.326416015625, 0.62646484375, -0.10015869140625, -1.6923828125, -0.72119140625, 0.2078857421875, -1.1953125, 0.54248046875, -0.5869140625, -1.3388671875, -0.400146484375, -0.99169921875, 0.11785888671875, -0.1934814453125, 0.8369140625, 2.75, 0.363037109375, -0.093017578125, -0.91064453125, -0.2054443359375, -0.0587158203125, 0.68115234375, -0.82763671875, -0.68798828125, 0.5439453125, 1.005859375, -0.265380859375, -1.2275390625, 0.468994140625, -1.2763671875, -0.7255859375, -0.430419921875, 0.442138671875, -0.494873046875, 0.1439208984375, 0.1593017578125, 0.370361328125, -0.254638671875, -1.3525390625, 0.77490234375, 0.124267578125, -0.50341796875, 0.85791015625, 0.36572265625, 0.75, 0.233642578125, -0.13525390625, -0.280517578125, 1.388671875, 0.259033203125, 0.84619140625, 2.45703125, -0.5947265625, -0.5361328125, 0.51123046875, 0.1954345703125, 0.998046875, 0.16552734375, -0.474853515625, 0.1959228515625, 0.265380859375, -0.51806640625, -1.4169921875, 0.361083984375, -0.1746826171875, 0.68994140625, 1.6748046875, -1.830078125, -0.047698974609375, 0.1920166015625, 0.828125, -0.050689697265625, -0.08221435546875, 1.0498046875, 1.515625, 0.8955078125, 0.44091796875, 0.703125, 0.87353515625, -0.187744140625, -0.69287109375, 0.144287109375, 0.050933837890625, 1.7822265625, 0.298828125, 0.394775390625, 1.1796875, 0.787109375, 0.97119140625, 0.708984375, 1.16015625, -0.86865234375, -0.4384765625, -0.15771484375, 0.70166015625, -0.397216796875, 0.1710205078125, 2.0, 1.01171875, 0.57958984375, 0.51171875, -0.421142578125, 0.415283203125, -0.1240234375, -0.5546875, 0.093505859375, 0.30322265625, -0.335205078125, -0.796875, 0.52001953125, -2.4140625, -0.0941162109375, 0.56640625, -0.1175537109375, 1.646484375, -0.1365966796875, 0.583984375, -1.1162109375, -0.05987548828125, 1.1767578125, -0.271728515625, -0.701171875, -1.0947265625, 1.2314453125, -0.994140625, -1.14453125, -0.67431640625, 0.5888671875, 0.72705078125, 0.386962890625, 0.6123046875, -1.5419921875, 0.426025390625, -0.09893798828125, 0.2198486328125, 0.230224609375, 0.84814453125, 0.3125, 0.488037109375, -1.8896484375, 0.33154296875, 0.1773681640625, -1.0048828125, 0.54931640625, -0.56982421875, -1.494140625, -0.10687255859375, -0.5390625, -1.7265625, -1.0322265625, 1.54296875, -1.0625, -1.046875, 0.479736328125, 0.468505859375, 0.83837890625, 0.309326171875, 0.2021484375, -1.4462890625, -1.2490234375, 0.85107421875, 0.323486328125, -0.68505859375, -0.85302734375, -0.853515625, 0.8212890625, -0.5087890625, -0.460205078125, 1.0263671875, -0.362548828125, 0.1978759765625, 0.50439453125, 0.053192138671875, -0.408935546875, 0.05181884765625, 0.456787109375, 1.0078125, 0.241455078125, -1.052734375, -0.5986328125, -1.5556640625, 0.29833984375, -1.48046875, 0.78857421875, 0.671875, 1.53515625, 1.4521484375, -1.1328125, 1.1767578125, 1.259765625, 0.389404296875, 0.431396484375, -0.1898193359375, -1.8291015625, 1.48828125, -0.6884765625, 0.49755859375, -1.1904296875, -2.078125, -0.9228515625, -1.1474609375, -0.7783203125, 1.7392578125, -1.6552734375, -0.5908203125, 1.0078125, -1.01171875, 2.111328125, 0.642578125, 0.051910400390625, -1.658203125, -1.21875, 0.40283203125, -0.340576171875, -3.3020973205566406e-05, -0.75732421875, 0.71435546875, 2.23828125, -0.08197021484375, -0.6201171875, 0.66162109375, -1.7568359375, 1.7216796875, 0.4267578125, 0.63037109375, -0.1561279296875, 0.56591796875, 1.0732421875, -0.1334228515625, -1.6884765625, -0.8896484375, -0.0040283203125, -1.236328125, 0.73828125, 1.6220703125, 0.89697265625, -0.54638671875, -1.11328125, 0.26171875, -1.2109375, 0.1881103515625, 0.08544921875, -0.7880859375, -2.53125, -0.8720703125, -0.5498046875, -0.55126953125, -0.25146484375, 2.12890625, 0.1448974609375, -0.4619140625, -1.5390625, 1.0947265625, -0.364501953125, -0.27783203125, -0.66259765625, -1.1572265625, -0.7060546875, 0.317626953125, 0.7333984375, -0.26953125, 2.103515625, -0.462890625, 2.748046875, 0.56982421875, -0.12225341796875, 0.71337890625, -1.4970703125, -0.1258544921875, -1.3076171875, 1.3251953125, -1.0087890625, -1.1728515625, 0.11181640625, -0.68115234375, -0.80517578125, 1.779296875, -1.369140625, -1.02734375, 0.6337890625, -0.26171875, 0.270751953125, -1.1142578125, 0.490234375, 0.113037109375, 0.350830078125, -0.128173828125, -0.05523681640625, -0.394775390625, -0.8212890625, 1.65234375, -0.218994140625, 0.8720703125, -1.2529296875, 0.7353515625, -0.306396484375, -0.81005859375, 0.736328125, -0.2164306640625, 0.091796875, -0.8056640625, 0.304443359375, 2.12109375, 2.466796875, 1.5205078125, -1.4775390625, 0.0516357421875, 0.64111328125, 0.60498046875, 0.9072265625, -1.2265625, -0.450927734375, -0.908203125, 1.1396484375, -0.27001953125, 2.3359375, -0.253662109375, 0.1451416015625, 0.023956298828125, -1.1181640625, 0.2880859375, -1.611328125, 0.45703125, -1.09765625, 2.251953125, -0.45703125, -0.09930419921875, -1.357421875, 1.4091796875, -0.26123046875, 0.339599609375, 0.98095703125, -1.033203125, 0.08270263671875, -1.29296875, 0.2340087890625, 0.181640625, 0.0196990966796875, -0.66259765625, 1.7138671875, 0.499755859375, 1.2919921875, 0.1431884765625, 1.3115234375, -0.0863037109375, -2.3359375, 0.93603515625, -0.031829833984375, 0.040008544921875, -1.455078125, 2.14453125, -0.90185546875, 0.783203125, 0.051513671875, 0.321533203125, -0.09515380859375, 0.86376953125, 0.027069091796875, 2.03125, 0.1317138671875, 0.2288818359375, 0.67919921875, 1.0791015625, 0.271240234375, -0.71923828125, 0.6171875, 1.5986328125, 0.2080078125, 0.8681640625, -0.68701171875, 0.003307342529296875, -0.09881591796875, -1.748046875, 1.14453125, -0.1314697265625, -0.0167388916015625, -0.024505615234375, 0.007511138916015625, 2.677734375, 0.1002197265625, 0.11212158203125, 0.37548828125, -1.052734375, -0.51806640625, -0.654296875, -1.1630859375, 1.0546875, 0.98681640625, 0.2066650390625, -1.0400390625, 0.39453125, -1.765625, -0.3447265625, -0.1634521484375, -0.6552734375, 0.82080078125, -2.197265625, 1.37109375, -1.4443359375, -0.0235137939453125, -0.07293701171875, 0.59619140625, -0.69091796875, -0.61474609375, -1.8759765625, 0.81982421875, 2.158203125, -1.267578125, 1.7890625, -1.2646484375, -0.919921875, -1.38671875, -0.703125, 0.91943359375, -1.34765625, 0.4541015625, 1.02734375, -0.2452392578125, -0.187255859375, -0.007022857666015625, 0.76611328125, 3.08984375, 1.1142578125, 0.33935546875, 0.54833984375, 0.5, 1.4345703125, -0.491455078125, -0.36865234375, -0.64501953125, -1.93359375, 1.818359375, 1.5869140625, -1.5419921875, 0.236328125, -1.044921875, -0.1295166015625, 1.349609375, 0.88720703125, 0.2235107421875, -0.11541748046875, -1.904296875, 0.1385498046875, -0.401611328125, -0.346923828125, -0.1551513671875, 0.492431640625, 0.51513671875, -1.8681640625, 0.50927734375, -0.537109375, 0.498779296875, -1.5927734375, -1.734375, -0.0279083251953125, 0.4150390625, -1.0244140625, 0.9765625, -0.1749267578125, 0.480712890625, -1.32421875, -0.22802734375, 1.2421875, 1.7890625, 1.447265625, -0.69384765625, 1.3837890625, -1.673828125, -0.44482421875, -0.2200927734375, -0.90185546875, -1.5146484375, -0.87353515625, -0.6337890625, -2.072265625, -1.474609375, 1.7138671875, -0.6826171875, -0.7490234375, 0.67724609375, -1.990234375, 0.01026153564453125, 0.130859375, -0.420654296875, 2.013671875, 0.392578125, -1.04296875, -0.75439453125, 0.98486328125, 0.221435546875, -0.9716796875, -0.166015625, 1.09765625, -0.40283203125, 1.083984375, 0.662109375, 2.427734375, 1.466796875, -0.86279296875, -0.5703125, -0.05224609375, -1.771484375, -0.783203125, 1.4775390625, -1.044921875, 1.0009765625, -0.51318359375, -0.7177734375, 1.7470703125, 1.79296875, 0.89990234375, -0.33544921875, 0.4970703125, 0.270751953125, -0.0618896484375, -0.1697998046875, -1.9814453125, 0.447509765625, 0.360107421875, -0.23681640625, 0.57470703125, 0.73828125, -0.59765625, -1.2314453125, -0.13671875, -2.55078125, 2.259765625, -0.6181640625, -0.50048828125, 0.66357421875, -1.0576171875, 0.60888671875, -1.939453125, -1.1318359375, -0.55712890625, -0.9482421875, 0.05029296875, -1.1728515625, 1.0009765625, 0.2276611328125, -1.103515625, 0.8583984375, 2.396484375, -0.0145721435546875, -0.113037109375, 0.05572509765625, -0.748046875, -0.322265625, 0.487548828125, 0.91015625, -0.5712890625, -0.11090087890625, 0.9384765625, -0.68896484375, 0.1927490234375, -0.251220703125, -0.09735107421875, 0.892578125, -0.0889892578125, 1.0361328125, 0.8759765625, -1.5810546875, 0.9345703125, -1.0810546875, -0.09210205078125, 0.36767578125, -1.658203125, 1.41015625, -0.73193359375, -1.6689453125, -0.8623046875, -1.0830078125, -0.438232421875, 0.474609375, 0.51416015625, -0.802734375, 0.55224609375, -0.8857421875, 0.8125, 0.1383056640625, -0.349853515625, -0.46484375, 2.53515625, -0.30615234375, 2.078125, 0.82470703125, -0.37890625, 0.98876953125, -0.1756591796875, -0.91162109375, -0.33203125, 0.88232421875, -1.359375, 1.4580078125, -1.3642578125, -1.984375, -0.40380859375, -1.587890625, 1.34765625, 1.2705078125, -0.421630859375, 0.93701171875, 0.96435546875, -0.1434326171875, -0.599609375, 1.2294921875, 1.85546875, -1.4443359375, 0.353271484375, -0.1492919921875, 0.97509765625, -0.98095703125, 0.74755859375, -0.77880859375, 0.68408203125, 0.560546875, 0.529296875, -0.5087890625, -0.245849609375, 0.77490234375, 1.1162109375, -0.1888427734375, -0.7802734375, 0.344482421875, -0.302490234375, -0.73974609375, -0.57861328125, -1.4306640625, 1.5087890625, 1.1396484375, -0.478271484375, -0.76708984375, 1.439453125, 1.9140625, 1.533203125, 0.75341796875, 0.01241302490234375, -0.857421875, -0.220458984375, -1.2314453125, -1.080078125, -0.1761474609375, -1.08984375, -0.58251953125, 0.28369140625, -1.091796875, 0.07818603515625, -0.73876953125, -1.37890625, 0.1453857421875, 0.14404296875, -0.01287841796875, 0.58984375, -1.380859375, -0.68701171875, 0.040374755859375, 1.5087890625, 0.58544921875, 0.30712890625, -0.256591796875, -0.440673828125, 0.05426025390625, -0.0438232421875, 0.7109375, 1.486328125, 1.0439453125, -0.5888671875, 0.12451171875, 1.0732421875, 0.5537109375, 0.72314453125, -0.7275390625, 0.1988525390625, 0.285888671875, -0.96875, 1.349609375, -2.8515625, -0.392822265625, 0.353271484375, -0.70947265625, 0.35888671875, -0.3251953125, -0.87744140625, -0.0528564453125, -0.493408203125, -2.201171875, -0.74658203125, -2.1015625, 0.90771484375, -1.0263671875, -0.64599609375, 0.2568359375, -0.0272216796875, 1.9541015625, 0.30517578125, -0.94287109375, 1.01953125, 0.56689453125, 0.7080078125, 0.86328125, -0.66455078125, -1.3603515625, -0.1588134765625, 1.1953125, 1.26171875, -0.69775390625, -0.02081298828125, -0.3515625, 0.7138671875, 0.0259552001953125, 0.357666015625, -1.7060546875, -0.189697265625, -0.0106353759765625, 0.059417724609375, 1.19921875, -1.3779296875, -0.1650390625, 2.828125, 0.810546875, -1.1015625, -0.6748046875, 0.2349853515625, -0.400390625, 1.173828125, 0.62890625, -1.2041015625, 0.63037109375, 0.3076171875, -0.7158203125, 1.49609375, 0.1966552734375, 0.7578125, -1.787109375, -0.386962890625, 1.02734375, 0.33154296875, -0.91796875, -0.3671875, 1.44921875, -0.1318359375, -0.83544921875, -0.54541015625, 0.159423828125, -1.39453125, 0.583984375, -1.4619140625, 0.5908203125, 1.740234375, 0.98193359375, 0.15234375, 0.66064453125, -1.119140625, 0.11981201171875, -1.8115234375, -0.134033203125, -0.1751708984375, -1.06640625, -1.6083984375, -0.5068359375, 1.1982421875, 0.68798828125, -0.55859375, -0.1795654296875, -1.859375, -1.0712890625, 1.4345703125, 0.01381683349609375, 0.312744140625, 0.8935546875, 0.5966796875, -1.060546875, -0.626953125, -0.71630859375, -0.458984375, 0.880859375, -1.4501953125, -0.82861328125, -0.09600830078125, 0.374755859375, -0.17919921875, 0.419921875, 0.732421875, 0.0538330078125, 0.58349609375, -0.2442626953125, 0.67529296875, -0.59326171875, 0.82763671875, 2.44140625, -1.3779296875, -0.8564453125, -0.56396484375, 0.370849609375, -0.050994873046875, 0.07293701171875, -0.1353759765625, -0.17919921875, -1.50390625, -0.284423828125, -0.2467041015625, -0.209228515625, 0.16845703125, 0.46435546875, 0.3642578125, 1.271484375, 0.1339111328125, -0.09222412109375, -1.1416015625, 0.327392578125, 0.35302734375, -0.12646484375, 0.5927734375, 0.93505859375, 1.4462890625, 0.6806640625, 0.11749267578125, 1.2958984375, 0.2216796875, -0.151611328125, 0.1431884765625, 0.1947021484375, 0.923828125, 0.55908203125, 1.07421875, 1.205078125, 0.365966796875, 1.6416015625, 2.189453125, -0.6455078125, -1.818359375, -0.90234375, 0.43505859375, 0.56640625, -1.04296875, -0.65185546875, 1.2431640625, -0.61669921875, 1.1064453125, -0.79833984375, -0.275390625, -0.1448974609375, 1.5712890625, 0.1136474609375, -0.61328125, -0.6318359375, -0.7822265625, 0.84521484375, 1.9521484375, -0.21923828125, -0.37744140625, -1.1298828125, -1.546875, -1.65234375, -0.07623291015625, -0.146240234375, -0.62158203125, 0.306884765625, -1.5322265625, 1.5927734375, -1.1337890625, 1.3232421875, 2.12109375, 0.13720703125, 0.82568359375, -0.272705078125, -1.0126953125, -0.01358795166015625, -1.10546875, 0.335205078125, 1.865234375, 0.54541015625, 0.9580078125, 1.240234375, -0.4130859375, -1.576171875, 0.031951904296875, -0.045745849609375, -0.0257415771484375, -0.54833984375, -0.74169921875, -0.501953125, -0.68115234375, 2.3046875, -0.435791015625, -1.2490234375, 1.4140625, -0.7265625, -1.052734375, 2.00390625, -0.48876953125, -1.4658203125, 0.9560546875, 0.8681640625, 0.876953125, 0.370849609375, -0.33056640625, -0.7646484375, 0.048675537109375, 0.3271484375, 1.15625, -0.1048583984375, -0.29052734375, 0.7431640625, -0.1837158203125, 0.61083984375, 1.9443359375, 0.9375, 0.681640625, -0.53955078125, -1.1484375, -0.763671875, 0.321044921875, -0.7099609375, -0.00856781005859375, -1.3974609375, 1.041015625, 0.80078125, -0.40771484375, -0.47021484375, -0.50732421875, -0.27978515625, 0.685546875, 1.0546875, 0.2626953125, -0.0177154541015625, -0.11651611328125, -0.0748291015625, -0.399169921875, -1.2080078125, -1.4716796875, 0.7607421875, 0.277587890625, 0.285888671875, -1.4287109375, -1.36328125, 0.6435546875, -1.1953125, -0.80419921875, -0.166259765625, -0.865234375, -2.712890625, 0.1480712890625, 0.1668701171875, -1.5068359375, 1.5, -0.078125, 0.38525390625, -0.3359375, 0.362548828125, -1.1572265625, 0.9287109375, 0.06640625, 0.61474609375, 0.417236328125, -2.2421875, -0.50634765625, 1.2919921875, 0.00533294677734375, 1.484375, -1.4384765625, 0.86865234375, -0.020782470703125, 1.57421875, -1.1533203125, -1.2236328125, 0.2117919921875, -0.70068359375, 0.27392578125, 1.0810546875, 0.1585693359375, -0.63134765625, -0.11090087890625, -0.75439453125, -0.95703125, -1.220703125, 1.1083984375, -0.26416015625, 1.3515625, -0.2283935546875, -1.2958984375, 0.16357421875, 0.371826171875, 0.1959228515625, 0.54931640625, 1.076171875, 0.251953125, -0.37451171875, -0.59814453125, -0.00457000732421875, 0.9208984375, -0.0894775390625, -0.4140625, -0.432861328125, 1.259765625, -0.90576171875, -0.6328125, -0.281494140625, 0.767578125, 1.8056640625, -0.0704345703125, -0.0235595703125, -1.11328125, -0.45849609375, -0.72998046875, -1.9609375, -0.2176513671875, -1.93359375, 0.71435546875, 1.7890625, 1.28125, -1.0185546875, 0.76904296875, 0.1429443359375, -0.888671875, 0.1580810546875, -0.86328125, -0.007335662841796875, 0.20849609375, -0.98779296875, 0.849609375, -0.97998046875, -0.29248046875, -1.748046875, -0.9248046875, -0.30126953125, 0.8408203125, -2.56640625, -1.615234375, 1.43359375, 1.3203125, 1.3740234375, 0.22607421875, 0.7705078125, -0.5693359375, 0.51416015625, -2.443359375, 2.255859375, -0.03570556640625, 0.1959228515625, -0.2293701171875, -0.64599609375, -0.33740234375, -0.493408203125, -0.826171875, 0.59375, -0.65283203125, 1.029296875, -1.59765625, -0.0916748046875, 1.3603515625, -0.52392578125, 0.654296875, 0.8349609375, -0.1444091796875, -0.2003173828125, 0.75244140625, 0.1334228515625, 0.284912109375, 0.126708984375, 0.338623046875, 0.0009312629699707031, -0.179931640625, 0.40283203125, -0.0675048828125, 2.337890625, -0.25146484375, 0.91943359375, 0.26513671875, 1.2294921875, 0.367919921875, 0.70068359375, -0.1580810546875, 0.46923828125, 1.75, -0.1019287109375, -0.09271240234375, 0.486083984375, 0.305419921875, 0.458984375, 0.2308349609375, -0.4912109375, -1.1796875, -1.4951171875, -0.17626953125, 1.2509765625, -0.3916015625, -0.342041015625, -0.3603515625, 1.1142578125, -1.5986328125, -0.2061767578125, -0.4716796875, 0.95458984375, 0.28955078125, -0.544921875, -0.294189453125, 0.28955078125, -0.04241943359375, -0.56982421875, 0.488525390625, -0.0943603515625, -0.393798828125, 1.5126953125, 0.658203125, 0.343017578125, 0.349853515625, 0.0892333984375, -1.228515625, -0.78125, -0.364013671875, -0.90576171875, 0.30615234375, -0.95703125, 1.9384765625, 1.6142578125, -1.671875, 1.0966796875, 0.0498046875, 0.348876953125, -1.2802734375, 0.86962890625, 0.92236328125, -0.8955078125, 0.55859375, 2.361328125, 0.11663818359375, 2.111328125, -0.1473388671875, 1.4423828125, -0.60791015625, -0.034637451171875, -0.43017578125, 0.1961669921875, 0.1158447265625, -1.0009765625, -0.5927734375, -0.26904296875, 0.1544189453125, -1.0732421875, 1.1845703125, -0.55029296875, 1.6171875, -1.2451171875, 0.33544921875, 1.767578125, 2.12109375, 2.912109375, -0.1444091796875, -0.83544921875, 0.5078125, 0.33740234375, -0.0120391845703125, 1.1826171875, -0.939453125, 0.5146484375, -1.3076171875, 0.6279296875, 1.421875, -0.343994140625, 1.115234375, 0.60791015625, 0.7080078125, 0.814453125, 0.529296875, -1.033203125, -1.93359375, 1.2373046875, -0.1611328125, -1.2958984375, 0.0977783203125, 0.27197265625, 0.2403564453125, -0.358642578125, 1.0322265625, 0.330810546875, -2.712890625, -0.3447265625, -0.45361328125, -0.578125, -0.837890625, 1.115234375, -0.4306640625, -1.4697265625, -0.493408203125, 0.40625, 0.474609375, -0.315185546875, 0.986328125, 0.43701171875, 1.26171875, 0.188232421875, -0.1966552734375, 0.6865234375, 0.8486328125, -0.08013916015625, -0.11456298828125, 0.344970703125, -0.9921875, -1.2587890625, -1.736328125, -0.998046875, 0.2012939453125, 1.3857421875, -0.2001953125, 0.5595703125, 0.09423828125, -0.759765625, 0.07318115234375, 0.78662109375, 1.013671875, -0.035675048828125, 1.248046875, 1.76953125, -1.39453125, 0.37548828125, -2.298828125, 0.68701171875, 0.8662109375, 0.79638671875, 0.09375, -0.3388671875, -0.2015380859375, 1.083984375, -0.2003173828125, 0.260498046875, -1.595703125, -0.96240234375, -0.1009521484375, -0.54931640625, -0.1439208984375, -1.2373046875, 0.84814453125, -0.87353515625, 0.11102294921875, -0.25537109375, -0.9794921875, -0.1304931640625, -1.478515625, -0.1705322265625, 0.0904541015625, -1.0537109375, -0.306640625, -1.310546875, -0.275390625, 1.658203125, 1.8134765625, -0.990234375, -0.463134765625, 0.440185546875, 1.529296875, 0.270751953125, 1.16015625, -0.70361328125, -0.53857421875, 0.0187225341796875, 0.33642578125, -0.023956298828125, -1.390625, -2.275390625, -0.1337890625, 0.035186767578125, 0.6005859375, -0.6865234375, -1.6259765625, -1.5791015625, -0.016754150390625, 0.462890625, -0.00353240966796875, -0.986328125, -0.85986328125, -0.170654296875, 1.6669921875, 0.8935546875, -0.1959228515625, -0.41796875, 1.537109375, -0.85791015625, 0.88330078125, 0.7041015625, -1.1240234375, -1.8349609375, -0.74560546875, -1.134765625, 1.041015625, -0.06842041015625, -0.7705078125, 1.1552734375, -0.60546875, 1.013671875, 0.1527099609375, -1.1494140625, 0.374267578125, -0.919921875, -0.826171875, 1.66796875, -1.0810546875, 0.54150390625, -1.52734375, -1.23828125, -0.609375, -0.5478515625, 0.07159423828125, 0.14892578125, 0.1900634765625, -1.7080078125, 0.2252197265625, 2.224609375, -1.2734375, 1.1044921875, -1.5341796875, 1.328125, 0.37255859375, 0.8837890625, 0.6064453125, -1.5205078125, -0.73193359375, -0.91650390625, 2.435546875, 0.970703125, 1.080078125, 1.103515625, 0.149658203125, -0.224365234375, 0.316162109375, 0.01009368896484375, 1.54296875, 1.3798828125, 2.455078125, -0.7490234375, 2.275390625, -0.470947265625, -0.197998046875, -0.6064453125, 0.122314453125, -0.221435546875, 0.039642333984375, 1.333984375, -0.93408203125, -1.2080078125, 0.35595703125, 0.466064453125, -1.2421875, -2.32421875, -0.62060546875, 0.045013427734375, 1.01171875, -1.3349609375, -0.239013671875, -0.60302734375, -2.47265625, 2.5234375, 0.455810546875, -1.2412109375, -1.19921875, 1.41015625, -0.6845703125, 0.68994140625, -1.0244140625, 1.5283203125, 0.040008544921875, 0.326416015625, 0.5849609375, -0.91552734375, -2.4375, -0.01507568359375, 0.2374267578125, 1.302734375, 0.7685546875, 0.2213134765625, -0.72412109375, 0.10595703125, -0.24267578125, -2.23046875, -0.61474609375, -0.2142333984375, -0.296630859375, 0.2064208984375, -1.4287109375, 0.228271484375, -0.60107421875, -0.521484375, 0.04510498046875, 0.69970703125, -0.71826171875, 1.0078125, 0.4384765625, -0.71142578125, -0.92626953125, -1.4658203125, -1.5126953125, -0.45263671875, 0.515625, -0.454345703125, -0.1474609375, 0.1787109375, 0.3291015625, -0.076904296875, -0.245361328125, -1.1845703125, -1.1943359375, 0.91064453125, 1.013671875, -0.1346435546875, -0.6142578125, -1.70703125, -1.3056640625, -1.3623046875, 1.6982421875, 1.6279296875, 1.26171875, -1.060546875, -1.4384765625, 0.99755859375, 0.2890625, 1.580078125, 0.55419921875, 1.6416015625, -0.4931640625, 0.1685791015625, -0.27294921875, 1.7919921875, -0.61865234375, -0.17138671875, 1.32421875, -0.99609375, -0.057098388671875, -0.0243072509765625, -0.951171875, -0.732421875, -0.76171875, 0.1605224609375, -0.86376953125, 0.09649658203125, -0.0350341796875, -0.3408203125, 0.256103515625, -1.9091796875, -0.31640625, 0.1290283203125, 1.517578125, -0.9716796875, 0.77392578125, -2.30078125, -1.01171875, 0.39990234375, -0.88330078125, 0.701171875, -0.235595703125, 0.923828125, 1.2216796875, -0.88525390625, 0.64404296875, -1.3671875, 0.05975341796875, -0.33837890625, -0.521484375, -0.439453125, -0.054046630859375, 0.55908203125, -0.95654296875, -1.0751953125, -0.5732421875, 1.291015625, 0.2452392578125, 0.424560546875, 0.0240631103515625, -1.07421875, -2.470703125, -0.86181640625, -1.1240234375, -0.787109375, -1.6357421875, -0.10040283203125, 1.2236328125, 0.6240234375, 1.521484375, 0.389892578125, 0.73828125, 0.051177978515625, -1.56640625, -1.068359375, 0.83203125, -0.6328125, -0.27392578125, 1.2861328125, -0.112060546875, -0.59814453125, -2.314453125, -0.159423828125, 1.4755859375, -0.546875, 1.728515625, -1.7021484375, 0.279052734375, 0.495361328125, 0.444580078125, -2.109375, 1.2509765625, 0.50830078125, 0.311767578125, -0.003631591796875, -0.269775390625, 0.52197265625, -1.8115234375, 0.5302734375, -0.08172607421875, 1.0771484375, -1.9423828125, 1.876953125, 1.0322265625, -0.78662109375, 0.410888671875, 0.308837890625, 0.36181640625, 1.1455078125, -0.8828125, 0.1754150390625, 0.13427734375, 1.0126953125, 2.2109375, 0.123291015625, 0.311767578125, 1.578125, 0.2890625, 1.67578125, -0.2861328125, 0.047149658203125, 0.3408203125, 0.60302734375, 1.0810546875, -1.07421875, -0.95751953125, 0.182861328125, -0.4443359375, 2.03125, 1.6083984375, -0.1951904296875, -0.8203125, 0.05126953125, 0.4765625, 0.52392578125, 1.931640625, -1.5576171875, -1.2890625, 0.5185546875, -0.93701171875, 2.173828125, -1.0654296875, 0.150390625, -0.00867462158203125, 0.39208984375, -1.791015625, 0.7705078125, 0.129638671875, 0.69091796875, -1.6025390625, 0.95654296875, -1.7744140625, -0.6943359375, -0.415283203125, -0.89697265625, 1.2734375, 0.82568359375, 1.8447265625, -0.10614013671875, 1.2109375, 1.1533203125, -1.1240234375, 0.3681640625, -0.56396484375, 0.74267578125, -1.23828125, 0.209716796875, -1.431640625, 0.50390625, 1.9951171875, 1.3330078125, 0.27783203125, 0.673828125, -1.21875, 1.375, -0.8134765625, 0.2098388671875, -0.291015625, 0.88134765625, -0.1092529296875, -0.2286376953125, 1.5947265625, 0.5341796875, 0.334716796875, 0.25830078125, -0.08160400390625, -0.5595703125, -1.357421875, 0.2015380859375, -1.12890625, -0.06744384765625, -0.31201171875, -1.3056640625, 0.28662109375, 0.5908203125, -2.38671875, -0.2120361328125, 1.095703125, -0.054595947265625, 0.63818359375, 0.77978515625, 1.2099609375, 1.935546875, 0.338623046875, -0.336181640625, 0.56103515625, 0.291015625, -1.2138671875, -0.432373046875, -1.068359375, -1.140625, 1.8603515625, 1.728515625, 0.69677734375, 0.59130859375, 1.900390625, -1.181640625, 0.09771728515625, 1.2734375, -1.7353515625, 0.29443359375, 1.0986328125, 0.73974609375, 0.2462158203125, -1.5283203125, 0.492431640625, -1.1943359375, 1.37109375, 0.54931640625, -2.3984375, -0.568359375, -2.146484375, -0.50537109375, -0.19775390625, 0.5478515625, -1.3369140625, 0.50634765625, -0.0259552001953125, -0.861328125, 2.087890625, -0.5615234375, -0.5380859375, -1.6337890625, 0.39697265625, -0.96923828125, 0.3984375, -0.380859375, 0.050811767578125, 0.060821533203125, 1.5048828125, -0.50830078125, -1.5625, -2.01171875, 0.34033203125, -1.8740234375, -0.986328125, 0.490966796875, -0.2264404296875, 1.1259765625, -0.765625, -1.4501953125, -0.1513671875, -0.049560546875, 1.0390625, -0.293212890625, 0.5751953125, 0.33203125, 0.86474609375, 0.68603515625, 1.0419921875, 0.06890869140625, -0.2320556640625, 1.142578125, 0.86083984375, 2.08203125, 0.2335205078125, -0.7412109375, -0.2388916015625, -1.4501953125, -0.5947265625, -0.299560546875, -1.1640625, 0.82080078125, -0.11041259765625, 0.50537109375, 0.1341552734375, 0.18896484375, -0.9111328125, 0.17333984375, -1.6728515625, 0.716796875, -0.42822265625, 0.3203125, 0.162109375, 1.3232421875, 0.54541015625, 0.84228515625, -1.578125, -1.0595703125, -1.6083984375, -0.08416748046875, 0.6474609375, 0.560546875, 0.208251953125, -0.509765625, 0.19921875, -1.6669921875, -0.4873046875, 1.3466796875, -0.1788330078125, -1.0732421875, -0.2802734375, 1.2607421875, -0.379638671875, 0.18603515625, 1.7822265625, -1.1162109375, 1.017578125, -0.345703125, -0.0032253265380859375, 0.397705078125, 0.3564453125, 0.5234375, -1.49609375, 0.95458984375, 0.10638427734375, 0.4677734375, -1.5390625, -0.87646484375, 0.34033203125, 0.2763671875, 0.40185546875, -1.04296875, -0.49755859375, -1.7958984375, 0.266357421875, 1.3935546875, 0.59765625, -0.05499267578125, -0.33349609375, -0.09722900390625, 1.5498046875, 1.482421875, 0.47021484375, 1.07421875, -0.52587890625, 0.04248046875, -1.17578125, -0.5634765625, 0.6494140625, 0.56201171875, 0.560546875, -0.088623046875, 1.013671875, 0.2440185546875, 1.33203125, 1.4462890625, 1.703125, -0.11163330078125, -0.0450439453125, -1.05078125, 0.9853515625, -1.28125, 0.420654296875, 0.66796875, -0.7470703125, -2.255859375, 0.2196044921875, -0.93115234375, 1.2265625, -1.845703125, 2.49609375, -0.84130859375, 0.60498046875, -0.31689453125, -1.119140625, 1.291015625, 1.1455078125, -0.07806396484375, -0.6337890625, -0.740234375, 0.513671875, -1.427734375, 0.322998046875, 0.62451171875, 0.425048828125, -0.35107421875, 0.2293701171875, 1.224609375, -0.6259765625, -0.6318359375, -1.0341796875, 0.7470703125, -1.115234375, -1.1279296875, 0.767578125, -0.7373046875, 0.75390625, 0.464111328125, 1.17578125, 0.198486328125, -1.67578125, 0.214599609375, -0.69970703125, -1.01171875, 0.75634765625, -0.783203125, 0.177734375, 1.76171875, 1.029296875, -0.07818603515625, 1.1689453125, 0.050201416015625, 0.179443359375, -1.3330078125, -0.912109375, -0.182861328125, -0.1331787109375, -1.9677734375, 1.380859375, 0.061676025390625, 0.81689453125, -0.77734375, -2.33203125, 0.35595703125, 1.0595703125, 0.431884765625, -0.2216796875, -0.45263671875, -1.5263671875, -0.57470703125, -0.42431640625, 0.80615234375, -0.6953125, -0.85546875, -0.08721923828125, -0.150146484375, -1.2060546875, 1.59765625, 0.448974609375, 0.311767578125, -0.17578125, 1.3037109375, -0.4267578125, -0.479736328125, -0.6279296875, 0.8076171875, 2.2421875, 1.64453125, 0.9033203125, -1.50390625, -0.783203125, 1.6533203125, 1.537109375, -0.002376556396484375, -0.120849609375, -0.052825927734375, -0.21435546875, -0.312744140625, -0.26611328125, 0.89453125, 0.2099609375, -0.9306640625, 0.82177734375, 1.5107421875, -0.658203125, 0.333251953125, 0.529296875, 0.806640625, 0.0204315185546875, 0.41455078125, 0.1541748046875, -0.92626953125, 0.1949462890625, -2.095703125, -2.25390625, 0.1370849609375, -0.857421875, 0.970703125, 1.232421875, -1.974609375, -0.6533203125, -1.1025390625, 1.19140625, -0.412109375, -1.2470703125, 1.0830078125, 0.89306640625, 1.1552734375, -0.28125, -0.09521484375, -1.0400390625, -1.30078125, 1.201171875, 0.94921875, -0.5849609375, -0.390869140625, 1.1845703125, 0.99755859375, 0.7529296875, -0.12249755859375, 1.1455078125, 0.2425537109375, 0.57373046875, 0.89892578125, 0.28515625, 0.042816162109375, -0.403564453125, 0.3369140625, -1.62890625, -0.0870361328125, 1.2861328125, 0.10345458984375, 0.19140625, 1.248046875, -0.958984375, -1.591796875, -0.2061767578125, -0.970703125, 0.35009765625, -0.246826171875, 0.37841796875, 0.330322265625, 0.172607421875, 0.6142578125, -0.4091796875, -0.273193359375, -0.8310546875, 0.2491455078125, -1.2548828125, 0.5712890625, 0.68701171875, 1.1953125, -0.943359375, -0.375244140625, 1.1669921875, -0.77392578125, 1.048828125, -0.75927734375, 0.015655517578125, -0.3173828125, -0.475341796875, 0.41796875, -0.181640625, 1.7802734375, 1.6982421875, -1.06640625, 0.798828125, -0.1912841796875, 0.6904296875, -0.734375, -1.775390625, -0.6845703125, 0.72021484375, -0.06646728515625, 0.1573486328125, 1.6103515625, -1.5361328125, 1.1669921875, -2.18359375, 1.0517578125, 0.38330078125, -0.170654296875, -0.921875, 2.228515625, -1.21484375, -0.73876953125, 1.3564453125, -0.890625, 1.2490234375, 1.712890625, 0.83251953125, -1.037109375, -0.42431640625, 0.031646728515625, 0.958984375, 0.84619140625, 1.451171875, 0.267333984375, -0.1845703125, -0.3662109375, 2.794921875, -0.8388671875, -0.69775390625, -1.35546875, -0.427490234375, 0.337646484375, -1.1611328125, 1.0927734375, 0.7197265625, -1.0361328125, 0.07611083984375, -0.059112548828125, -1.4599609375, -1.1572265625, -0.63134765625, -1.7412109375, -1.1162109375, -1.1142578125, 0.1710205078125, 0.6474609375, -0.88037109375, -0.328369140625, 0.203857421875, -0.72216796875, -1.462890625, 1.7490234375, -0.0149993896484375, -0.5078125, 1.123046875, 0.06451416015625, 0.422607421875, -0.95263671875, 0.7744140625, 0.498291015625, 0.63525390625, 0.73828125, -0.7578125, -1.19921875, 1.033203125, 0.67138671875, 0.58740234375, 0.59814453125, -1.498046875, -0.0367431640625, -0.728515625, -2.4375, -1.267578125, 0.020263671875, 2.033203125, 1.3046875, 1.8310546875, -1.423828125, 0.1986083984375, -1.2822265625, -0.208251953125, 0.61376953125, 0.4736328125, -1.0419921875, 1.1845703125, -0.9130859375, 0.366455078125, 0.35546875, 0.22216796875, -1.7587890625, 0.360595703125, -2.013671875, -0.5556640625, -0.11090087890625, -0.7314453125, -0.59716796875, -0.40869140625, 0.681640625, -1.1943359375, -0.3330078125, -0.3388671875, -2.7265625, 0.890625, -1.8935546875, 2.0859375, 0.525390625, 0.48486328125, 1.505859375, 0.296875, -1.0224609375, 0.85546875, -1.5439453125, 0.201171875, 0.4296875, -0.94482421875, -1.3662109375, -2.37890625, -2.107421875, -0.70947265625, -0.1490478515625, 0.51953125, -0.06781005859375, 0.79150390625, -0.407958984375, 0.06304931640625, -0.09918212890625, 1.8955078125, 0.02874755859375, 1.94140625, -1.033203125, -2.23828125, 0.57373046875, 1.2548828125, -2.0546875, 0.5869140625, -0.463134765625, -1.9228515625, -0.12481689453125, -0.80419921875, 0.50439453125, 1.419921875, -0.5009765625, -0.6982421875, -1.068359375, -0.253173828125, 0.69580078125, -0.1275634765625, 0.75, 0.029296875, -0.80712890625, 0.495361328125, 1.3447265625, 1.1025390625, -0.80419921875, -1.103515625, 0.186767578125, -1.2939453125, 0.41259765625, -0.54052734375, -0.1883544921875, -0.32421875, -1.619140625, 0.1864013671875, -0.5302734375, -1.544921875, 1.1689453125, 0.8759765625, 0.1962890625, 1.255859375, 0.609375, -0.401611328125, -1.8837890625, -0.537109375, -0.18505859375, -0.953125, 0.36083984375, 0.56298828125, -0.445068359375, -0.0751953125, 1.13671875, -0.489990234375, -0.51025390625, 0.703125, -1.279296875, 0.304443359375, -0.3349609375, 2.267578125, -0.075439453125, -0.822265625, -0.1080322265625, -0.4033203125, 0.407470703125, 0.62451171875, 0.60693359375, 0.04412841796875, -0.828125, -0.38916015625, -1.1064453125, 0.058349609375, 0.024810791015625, -0.94580078125, 0.8564453125, 0.00591278076171875, 1.3330078125, 0.257568359375, -0.328369140625, -1.7978515625, -0.92138671875, 1.513671875, 0.244384765625, -1.21484375, -0.595703125, 1.34375, -0.7763671875, -0.7099609375, 0.826171875, 0.0775146484375, -1.9892578125, -0.392822265625, 0.8017578125, -1.7470703125, 0.146484375, -0.34326171875, 0.45703125, -0.49755859375, -0.48388671875, -0.2291259765625, -0.1702880859375, 0.47265625, 0.09576416015625, -1.765625, 0.88232421875, 0.0235137939453125, -0.798828125, 0.274169921875, -0.03216552734375, 1.9287109375, 0.29345703125, -0.7470703125, -2.17578125, 1.4765625, 1.7890625, -0.078125, 1.28125, 0.0479736328125, -0.332275390625, -2.908203125, -2.22265625, -0.499755859375, 1.4111328125, 0.435302734375, 0.11444091796875, -0.41162109375, 1.521484375, 1.75390625, -2.40234375, 0.0338134765625, 1.6435546875, 0.1492919921875, 0.91015625, -0.71533203125, 0.7763671875, 0.8125, -0.75341796875, -0.1307373046875, -0.07989501953125, -0.53759765625, 0.240966796875, -0.4814453125, -0.83447265625, -1.208984375, -0.68212890625, -0.059356689453125, 0.035797119140625, 0.014801025390625, -0.10052490234375, 2.0546875, 0.4833984375, -1.177734375, 0.7734375, -2.64453125, 1.810546875, 0.62158203125, 2.0234375, 1.017578125, -0.57373046875, -0.73876953125, 0.52978515625, -0.7177734375, -1.08203125, 0.333251953125, 0.453857421875, -1.7431640625, -0.79931640625, -1.017578125, 0.55224609375, 0.01190948486328125, 0.0087890625, 0.392822265625, -0.408447265625, -0.0701904296875, 0.28515625, 0.84765625, -1.14453125, 2.103515625, 0.392822265625, -0.609375, 0.90185546875, -0.37841796875, 0.409912109375, 0.0268402099609375, 0.383056640625, 2.3359375, 0.5927734375, -1.3212890625, 0.21630859375, -1.0517578125, 0.61279296875, -1.88671875, -1.2939453125, -0.314453125, 0.60888671875, 0.477294921875, -0.1849365234375, -1.572265625, -1.2373046875, -0.00650787353515625, 0.5966796875, -1.3056640625, -0.494140625, -0.93212890625, 0.85400390625, 0.040771484375, -0.900390625, 1.09375, 1.2099609375, 0.288818359375, 1.45703125, 0.10089111328125, -0.544921875, -2.2421875, 0.2276611328125, -0.72802734375, 0.8642578125, 0.84228515625, 0.310302734375, 0.71923828125, 1.8955078125, 0.7880859375, 1.0361328125, 0.5478515625, -0.9609375, -1.7333984375, 0.72607421875, 1.396484375, 0.2384033203125, -0.171630859375, 1.5830078125, -1.419921875, -0.1846923828125, 0.4560546875, -0.37255859375, -0.5869140625, -0.8818359375, 0.91943359375, 0.496826171875, -0.45068359375, -0.279052734375, 0.426513671875, -0.433837890625, 0.380126953125, 1.501953125, -0.484130859375, 1.0, -0.388427734375, 0.6298828125, -0.84228515625, -1.009765625, 0.07586669921875, -0.2335205078125, -0.1466064453125, 0.8642578125, 0.744140625, 1.857421875, -0.00675201416015625, -0.256103515625, 0.11956787109375, -0.1409912109375, 0.6728515625, -0.01145172119140625, -1.1923828125, 0.861328125, -0.163330078125, -0.239990234375, -1.3896484375, -0.66552734375, -2.072265625, 1.560546875, -0.06060791015625, 0.140869140625, 0.45263671875, -1.7666015625, -1.72265625, -0.181640625, -0.822265625, 0.65576171875, -1.0830078125, -0.029022216796875, 0.28515625, -1.1396484375, -0.12310791015625, 0.276611328125, 0.95361328125, -1.8095703125, 0.97900390625, 0.66748046875, 0.16455078125, -0.195068359375, 0.7451171875, -1.5341796875, -1.21484375, -1.193359375, -2.830078125, 0.80078125, -0.36181640625, -1.4052734375, 1.255859375, 0.435302734375, 0.484619140625, 0.036346435546875, 0.1612548828125, 0.345703125, -0.48681640625, 1.0029296875, 0.123779296875, -0.61474609375, 1.9619140625, -0.7314453125, 0.44580078125, -0.060211181640625, 0.72119140625, 1.546875, -1.27734375, -0.8349609375, -0.4130859375, -0.77490234375, -0.376708984375, 0.79736328125, -0.399169921875, 0.01708984375, 0.11083984375, -0.400634765625, -0.103515625, -2.04296875, 0.60546875, 0.085205078125, 1.2802734375, -0.69091796875, 1.173828125, -1.953125, 1.748046875, 1.1572265625, 0.356689453125, -0.79638671875, -0.79443359375, -0.84326171875, -1.3828125, 0.80126953125, 1.2255859375, 2.908203125, 0.61328125, -1.0673828125, -0.9755859375, 0.79931640625, -0.99169921875, -0.7900390625, 0.66552734375, 1.47265625, 0.2425537109375, -0.65380859375, 0.25830078125, 1.3125, 1.267578125, 2.46484375, -1.1416015625, -0.865234375, 0.498779296875, -1.001953125, -0.4443359375, 0.19970703125, -0.141845703125, -0.4052734375, -0.76513671875, 1.63671875, -1.7724609375, 0.2235107421875, 0.79638671875, -1.8291015625, 0.3564453125, -0.86669921875, 0.97802734375, 0.006160736083984375, -0.1927490234375, -0.4697265625, 1.560546875, -0.91015625, -0.2073974609375, -0.78759765625, 2.169921875, -0.1533203125, -0.3828125, -0.11981201171875, -1.412109375, 0.66650390625, -0.89013671875, 0.74951171875, 0.82080078125, 0.96826171875, 1.0234375, -0.76171875, -0.95751953125, 0.60107421875, 0.85693359375, 1.0361328125, -0.5029296875, -2.15625, 0.0799560546875, -1.0615234375, 0.5693359375, -0.85546875, -0.8759765625, -0.7998046875, 0.9189453125, -0.11187744140625, -0.1502685546875, -0.416015625, -0.389404296875, -0.9951171875, 0.5908203125, 1.4814453125, 0.50146484375, 0.2529296875, 1.2197265625, 0.1600341796875, 0.27880859375, -0.607421875, -0.9365234375, -0.359619140625, 0.438232421875, 0.131591796875, -1.52734375, 0.525390625, 0.90966796875, -0.435546875, 0.425048828125, 0.332275390625, -0.1435546875, 0.0887451171875, 0.8427734375, -0.7724609375, 0.48193359375, -0.27197265625, 1.2255859375, 1.0400390625, -0.342529296875, -0.68115234375, -0.54833984375, -0.06689453125, 0.460693359375, -1.2900390625, -0.8408203125, -0.9345703125, -0.85498046875, -1.09765625, 0.86474609375, 0.378173828125, 0.1873779296875, -0.445068359375, 0.30712890625, -1.9873046875, 0.302734375, -0.294677734375, 1.7568359375, 0.02423095703125, 0.99609375, -0.51708984375, 0.69775390625, -0.2098388671875, 2.193359375, -0.80322265625, 0.59521484375, -0.37060546875, 0.261962890625, 1.21875, -1.220703125, 0.84033203125, -0.266357421875, -1.146484375, -0.73583984375, -0.487548828125, 0.2005615234375, 0.10693359375, 0.498779296875, -0.046173095703125, -0.6494140625, 1.1708984375, -0.08233642578125, 0.227783203125, -0.1458740234375, -1.435546875, -0.2225341796875, 0.62841796875, 0.465576171875, 0.2420654296875, -0.1461181640625, -0.22021484375, 0.61376953125, -1.0537109375, -1.6044921875, -0.41552734375, -0.78564453125, 0.31591796875, 1.98828125, -0.54248046875, -0.12359619140625, -1.064453125, -0.041259765625, -0.5439453125, 0.53955078125, 0.49658203125, -0.43115234375, -0.712890625, -0.84228515625, -0.1072998046875, 0.830078125, -0.0193939208984375, 0.460205078125, -0.1470947265625, 0.88818359375, -2.0234375, -1.1494140625, 1.0322265625, 1.111328125, -1.1845703125, 0.88525390625, -1.77734375, -0.19775390625, -0.1959228515625, 1.6962890625, 0.80615234375, -2.2734375, -0.73974609375, 0.1949462890625, 0.857421875, 0.01885986328125, -1.4912109375, 0.0270538330078125, 0.060089111328125, -0.125732421875, -1.0390625, -0.2220458984375, -0.64453125, 0.103759765625, 0.428466796875, -0.240234375, -0.87548828125, -0.336181640625, -0.054229736328125, 0.38232421875, -0.497802734375, -0.09918212890625, -1.326171875, -0.72119140625, -1.0927734375, 0.76708984375, -1.1572265625, 0.51220703125, -1.119140625, 0.467041015625, 1.3818359375, 0.50439453125, -1.044921875, 0.1485595703125, -0.61181640625, -0.09423828125, -0.50146484375, 0.51904296875, 0.6416015625, -0.22607421875, 0.0176239013671875, 1.1083984375, -0.375732421875, 0.79052734375, -0.4404296875, -0.12103271484375, 0.2047119140625, -0.69140625, -0.87744140625, -0.2181396484375, 1.630859375, -0.305908203125, -0.429931640625, 0.5224609375, -0.9677734375, -0.7138671875, -0.81689453125, -1.9892578125, 0.67333984375, 0.0204315185546875, -1.0546875, -1.13671875, -0.90283203125, 0.52001953125, 1.365234375, 0.050537109375, -1.6640625, 0.198974609375, -1.849609375, -1.2255859375, 0.61865234375, 0.353515625, -0.6279296875, 0.3818359375, -0.010284423828125, -0.2890625, 0.48583984375, 0.5927734375, -1.466796875, 1.611328125, 1.1201171875, 0.462646484375, -1.96875, 1.025390625, 0.156982421875, -0.24755859375, 0.52734375, 1.4306640625, 1.15625, -0.4697265625, 0.0997314453125, 0.6220703125, 1.5625, -0.712890625, 0.83642578125, -0.9072265625, -1.8671875, 0.3544921875, 0.45947265625, 2.912109375, -0.450927734375, 0.27197265625, -0.76171875, -1.666015625, -0.5380859375, -1.4609375, -1.7177734375, -0.1641845703125, 0.97509765625, -0.4365234375, 0.053924560546875, 1.2412109375, -1.83203125, 0.228271484375, 1.1337890625, 1.2109375, 2.185546875, 1.2373046875, -1.5654296875, -0.24951171875, 0.10223388671875, 0.280517578125, -0.72998046875, 0.9326171875, -0.81884765625, -0.151611328125, -1.494140625, 0.26123046875, -1.0830078125, 1.52734375, -0.109619140625, -1.3427734375, -0.59619140625, -0.939453125, -1.2841796875, 1.3515625, -0.311767578125, -0.3857421875, 0.892578125, 0.41650390625, -0.378662109375, 0.9814453125, 0.4365234375, 0.517578125, -0.32763671875, -1.0458984375, -0.434326171875, 1.87890625, -0.55615234375, 1.8515625, 0.1319580078125, -0.354248046875, 0.77001953125, -1.3125, 0.63671875, 0.2744140625, -0.65625, -0.415283203125, 0.393798828125, -0.09832763671875, 0.2113037109375, 1.5, 0.3916015625, 1.318359375, 0.37353515625, 0.5712890625, -0.52734375, -0.33251953125, -0.400146484375, -1.041015625, 2.361328125, -1.5888671875, -0.147216796875, -0.347900390625, -0.52880859375, -0.65625, -0.105224609375, 0.7626953125, 0.036102294921875, 0.0269927978515625, -0.46728515625, 0.60205078125, 1.18359375, -0.12255859375, 0.40380859375, -1.859375, 0.276611328125, 0.96337890625, 1.0888671875, 0.89892578125, 0.8037109375, -1.724609375, 0.73193359375, -0.6787109375, -0.69189453125, -0.241455078125, 0.1614990234375, -0.97314453125, 0.254638671875, -0.038299560546875, -0.2239990234375, -0.86279296875, 0.08514404296875, -0.56396484375, 0.2509765625, -1.9150390625, -0.198486328125, 0.0849609375, 0.8486328125, 0.63916015625, -2.75390625, 0.036224365234375, -1.3857421875, -0.326904296875, -0.98291015625, -0.125732421875, 0.2161865234375, -0.74853515625, 1.04296875, 0.70361328125, -1.087890625, -1.60546875, -1.1083984375, -0.95556640625, 0.12457275390625, 0.46728515625, -0.4775390625, -1.46875, -0.61572265625, -0.2666015625, 0.61962890625, 0.79931640625, -0.09539794921875, -0.492919921875, -1.1044921875, 1.1884765625, 0.611328125, 0.0175018310546875, 1.4169921875, 0.12286376953125, 0.270751953125, -0.4765625, -0.72900390625, -0.16552734375, 0.96142578125, -1.458984375, -0.751953125, -0.12353515625, -2.267578125, -1.0732421875, -0.53076171875, 0.2398681640625, 0.318359375, -0.79248046875, 2.25, 1.07421875, 0.6884765625, 1.1455078125, 1.0478515625, 1.37890625, -0.1929931640625, -1.138671875, -0.51708984375, -0.70556640625, 0.10601806640625, 0.994140625, 1.3466796875, -0.4951171875, -0.693359375, 0.309326171875, 1.7314453125, -0.00991058349609375, -0.95849609375, -1.0751953125, -2.00390625, -0.54052734375, 0.69287109375, 1.166015625, 1.37109375, 0.450927734375, -2.076171875, -0.78076171875, 0.142578125, 1.3583984375, 1.4736328125, -1.0205078125, 1.2490234375, -0.62060546875, -0.6630859375, 1.0947265625, -0.451904296875, 0.64453125, -1.2265625, -0.32275390625, 1.4580078125, 0.11737060546875, 0.53271484375, -0.234375, 1.8720703125, 1.6708984375, -1.3984375, 1.865234375, 0.84814453125, 0.152587890625, -1.591796875, -0.54150390625, 2.015625, 0.78564453125, 1.1201171875, -1.1064453125, 0.391845703125, 0.4462890625, 0.055450439453125, 0.31787109375, -0.55859375, 1.5625, -0.7939453125, -0.0994873046875, -0.87939453125, -0.96142578125, 1.2275390625, -0.65283203125, 2.82421875, -1.9091796875, -1.677734375, -0.290283203125, -0.59765625, 1.521484375, -1.638671875, -0.22900390625, 1.0390625, 1.1748046875, 0.95703125, 0.2255859375, 0.341552734375, -0.034271240234375, -0.646484375, 1.7080078125, -0.5390625, -0.6484375, 0.88720703125, -0.21875, 0.1873779296875, 1.17578125, -1.2294921875, 2.345703125, -0.2529296875, -0.177734375, -0.3603515625, 0.80322265625, 0.09857177734375, -0.5009765625, 0.56982421875, 0.352783203125, -0.80029296875, 0.6728515625, 0.7314453125, -0.50732421875, 0.0163116455078125, -1.451171875, 1.5703125, -0.54052734375, -0.0965576171875, 0.6875, -0.7744140625, 0.4267578125, -1.77734375, -1.015625, -0.9521484375, -0.7900390625, -0.77294921875, 0.98828125, 0.57373046875, -0.53125, 1.625, -0.4482421875, 0.1600341796875, -1.458984375, 0.04034423828125, -2.953125, 0.55029296875, -0.41796875, -1.5888671875, 0.55078125, -0.18115234375, 0.172119140625, 0.1151123046875, -0.050994873046875, 0.59814453125, 0.00946044921875, 0.78662109375, 0.73193359375, 0.58349609375, -0.44970703125, 1.228515625, 0.167724609375, 0.92724609375, -0.142578125, 0.83984375, 0.1214599609375, 0.953125, 0.7509765625, -0.447998046875, 0.8955078125, -0.0198974609375, 0.23681640625, -2.220703125, -0.83740234375, 0.07830810546875, 0.591796875, -0.049957275390625, 0.404541015625, 1.408203125, 1.0927734375, 0.76708984375, 1.3330078125, -0.701171875, 1.4453125, 1.2998046875, 0.98876953125, -1.265625, 0.177490234375, 0.5888671875, -1.4716796875, -0.30712890625, -1.541015625, -0.12042236328125, 1.9794921875, -0.0753173828125, 0.919921875, -1.201171875, -1.189453125, -0.669921875, -0.546875, 0.0110015869140625, -0.91162109375, -0.30126953125, 0.468017578125, 0.76025390625, 1.173828125, 1.591796875, -0.9287109375, -0.058135986328125, -0.68701171875, 0.62158203125, -0.82080078125, -0.7060546875, -0.55322265625, -1.1572265625, 0.7578125, 0.24853515625, -0.294677734375, 0.046539306640625, -0.245361328125, -0.7421875, 0.294677734375, -1.0546875, -0.2088623046875, -0.458251953125, 0.482666015625, -0.64111328125, -0.28857421875, -0.79443359375, -0.45751953125, 0.845703125, 0.13330078125, -0.06317138671875, 0.382568359375, -0.0237579345703125, 0.6982421875, 0.53271484375, -0.30322265625, 0.1573486328125, -0.1444091796875, 0.8681640625, 0.32470703125, 1.3203125, 0.275146484375, -0.1861572265625, -1.1298828125, -0.79638671875, 0.451171875, -0.3134765625, 0.76806640625, -0.7861328125, -0.9580078125, -0.2071533203125, 0.86181640625, 1.68359375, 0.556640625, 0.3046875, 1.390625, -0.1270751953125, 0.1171875, 0.9267578125, -0.96142578125, -0.219482421875, -0.033233642578125, -1.3701171875, 0.493408203125, -1.298828125, 0.3583984375, -0.050079345703125, 0.6103515625, -0.046630859375, 0.1551513671875, 1.095703125, 1.8486328125, -1.412109375, 0.80517578125, -1.517578125, 0.1053466796875, -2.48046875, -0.06634521484375, -0.2354736328125, -0.2069091796875, -0.68603515625, -0.7490234375, 1.2119140625, 0.265869140625, 0.76904296875, -0.208251953125, -0.31591796875, 0.54541015625, -0.16796875, 0.09674072265625, 0.8251953125, -2.525390625, -0.1461181640625, 0.5029296875, 0.472900390625, 0.0872802734375, 1.0224609375, 1.513671875, -1.4697265625, -1.1064453125, -2.98046875, 0.841796875, -1.6171875, -0.1639404296875, 0.74951171875, -0.93310546875, 0.59521484375, -0.46875, -0.334716796875, 1.0888671875, 0.353271484375, 0.1015625, 0.128173828125, -0.381103515625, -0.2156982421875, 0.43359375, -0.290771484375, -0.79052734375, -1.56640625, -1.080078125, -0.382568359375, -0.173828125, -1.240234375, 0.08740234375, 1.1298828125, 0.3515625, -0.1810302734375, -0.94970703125, 1.2509765625, -0.03594970703125, 0.2015380859375, -0.2169189453125, 0.708984375, 0.62548828125, 0.6884765625, -0.07684326171875, -0.084716796875, 0.71044921875, -0.245361328125, 0.99560546875, -1.126953125, 0.263427734375, -1.05078125, 0.9365234375, 0.54541015625, -0.117919921875, 0.63818359375, -0.673828125, 0.02667236328125, 0.451904296875, -0.442626953125, -1.58203125, 0.56787109375, 0.71728515625, -0.396484375, -1.138671875, -0.98193359375, -0.830078125, 0.63134765625, 0.467529296875, 0.8896484375, -0.4189453125, -2.609375, -0.3525390625, -1.5947265625, 0.66796875, 0.9306640625, -0.76318359375, -1.17578125, -0.96923828125, 0.6923828125, 1.109375, -2.10546875, -0.98583984375, 1.6181640625, 0.12451171875, -0.004791259765625, -0.032562255859375, 0.346923828125, -0.45751953125, 0.259765625, 0.8994140625, -1.2587890625, 0.798828125, 3.29296875, -1.5234375, -0.61962890625, 0.98876953125, -1.09765625, -0.353271484375, -1.14453125, 0.1275634765625, 0.0894775390625, -0.1439208984375, -0.9423828125, -1.240234375, 0.861328125, -1.1865234375, -0.6025390625, -0.53759765625, 0.26123046875, -0.53125, 0.07330322265625, -0.388916015625, -0.5341796875, -0.0772705078125, -0.482666015625, 0.5732421875, 0.2291259765625, -0.242431640625, 0.5068359375, -0.65283203125, -1.21484375, 1.94921875, -0.1956787109375, -0.427978515625, 0.8486328125, -1.5947265625, -0.58203125, -0.46630859375, -0.2626953125, -0.27197265625, -0.6748046875, -2.076171875, 0.2174072265625, 1.078125, -0.34912109375, -1.263671875, -2.05859375, -0.1639404296875, 0.84130859375, 0.68212890625, 1.8134765625, 0.345947265625, -0.015045166015625, -1.25, -0.6025390625, -0.047210693359375, -0.130859375, 1.00390625, -1.1484375, -0.52685546875, -0.60693359375, -0.51171875, -0.0860595703125, -1.2109375, -0.16552734375, 0.2193603515625, 0.1837158203125, -0.521484375, -0.022247314453125, 1.3212890625, 0.75390625, -0.446533203125, -2.962890625, 0.258056640625, -0.1583251953125, -0.07342529296875, -0.297607421875, 0.23095703125, 0.5634765625, 0.90966796875, 0.95556640625, -1.21484375, -1.6865234375, 0.497314453125, 0.0675048828125, 2.458984375, -1.765625, 0.89892578125, -1.4287109375, -1.0751953125, -0.92236328125, -0.303466796875, 0.36572265625, 0.88623046875, 1.525390625, -0.61865234375, 0.9560546875, 0.634765625, 0.63818359375, -0.07489013671875, -2.478515625, 1.1357421875, -0.79931640625, -0.323486328125, -0.50341796875, -0.107666015625, 1.263671875, -0.33154296875, -0.74755859375, 0.0102691650390625, -0.04034423828125, -0.4765625, -0.403564453125, -1.072265625, 0.7529296875, 0.5576171875, -0.79736328125, -0.340576171875, -0.436279296875, -1.701171875, -0.63623046875, -0.0762939453125, 0.154541015625, -0.82275390625, -0.0200347900390625, -0.57958984375, -0.865234375, -0.84619140625, 0.5947265625, -0.12091064453125, -2.10546875, 0.53759765625, -2.15234375, 1.1689453125, -1.259765625, -0.0006160736083984375, 0.35693359375, -0.059173583984375, 0.705078125, 0.310791015625, -1.609375, 0.033935546875, -1.146484375, 0.61962890625, 2.0, -0.73974609375, -0.3369140625, 1.201171875, 1.0458984375, -0.666015625, 0.712890625, 0.80126953125, -1.9541015625, 0.31005859375, 1.69921875, 0.37548828125, 1.623046875, -0.137939453125, 0.61279296875, 2.13671875, 0.5185546875, -0.427001953125, 0.197021484375, -1.009765625, 0.830078125, 0.57861328125, -0.328125, 0.1839599609375, -0.7333984375, -0.445556640625, -2.58984375, 1.1181640625, -0.58349609375, 0.1629638671875, 0.90673828125, 0.9365234375, 1.0166015625, 0.402587890625, -1.4482421875, -0.7353515625, 0.057159423828125, -0.4599609375, 0.387939453125, -0.0316162109375, -0.0941162109375, -1.060546875, -0.9482421875, 0.5341796875, 0.82421875, -0.055267333984375, -0.356201171875, -0.0016355514526367188, 0.76220703125, -0.0843505859375, 0.6728515625, -0.60693359375, -0.90869140625, -0.97412109375, 0.020538330078125, 1.513671875, -0.64306640625, -0.87548828125, 1.3876953125, 0.05548095703125, 0.324462890625, 0.412353515625, -1.3466796875, 0.96337890625, -1.09375, -0.304443359375, -0.1815185546875, 1.05859375, 0.7900390625, -1.015625, -0.274658203125, 1.2314453125, -1.1298828125, -0.8623046875, -0.72607421875, 0.14794921875, 1.7490234375, -0.65234375, -0.7294921875, -0.732421875, 0.1849365234375, -0.320556640625, 0.66357421875, 1.0263671875, 0.9384765625, 0.04498291015625, 0.5078125, 0.994140625, 1.30859375, -0.08978271484375, -1.2294921875, -1.1806640625, 0.270751953125, -0.06951904296875, 0.9794921875, -0.425537109375, 0.4375, 0.308349609375, 0.94580078125, -0.07867431640625, -0.7861328125, 0.86328125, 1.0302734375, 1.4326171875, -0.370361328125, -0.90380859375, 0.6591796875, -1.341796875, -0.80126953125, -0.321044921875, -0.2225341796875, 0.06744384765625, -1.4990234375, 1.4951171875, -0.59423828125, 1.8779296875, -0.87646484375, 2.119140625, -0.116455078125, -0.98681640625, -0.9736328125, 0.6142578125, -0.172119140625, -0.60791015625, -0.257080078125, -0.70361328125, -0.830078125, 0.9375, 0.306396484375, 0.064208984375, -0.85107421875, 0.75537109375, 1.4443359375, -0.325927734375, -0.43994140625, 0.71044921875, 0.58642578125, -0.4638671875, 1.1376953125, -0.55615234375, 0.054718017578125, 2.287109375, -0.332275390625, -1.8837890625, -0.489013671875, 1.8232421875, 0.490478515625, -1.6611328125, -0.41650390625, 0.347900390625, -0.32958984375, -0.84130859375, 0.2225341796875, 0.95166015625, 0.86474609375, -0.12109375, 0.002552032470703125, -0.77294921875, 0.892578125, 0.88427734375, 0.310302734375, 0.170654296875, -0.8017578125, 0.059844970703125, 0.76513671875, 0.2724609375, -0.335205078125, -0.6201171875, 1.1025390625, 0.1263427734375, 0.412841796875, -0.99462890625, 0.045928955078125, 0.319580078125, 0.12200927734375, -0.77392578125, -0.461181640625, 0.427978515625, 0.49169921875, -1.9658203125, 0.53759765625, -1.3173828125, 0.94140625, 1.271484375, -1.0224609375, 0.91357421875, 1.001953125, -0.400634765625, -0.4150390625, 0.049560546875, -0.0185699462890625, -0.54248046875, 0.78369140625, -0.7548828125, -0.900390625, 1.0849609375, -0.60693359375, 1.0751953125, 0.30126953125, 0.11798095703125, 0.60498046875, 0.5556640625, -0.76806640625, 0.7529296875, -0.4560546875, -1.0361328125, 1.2236328125, -0.094482421875, -0.383544921875, -0.53125, 0.91455078125, -1.162109375, -0.58349609375, 0.052764892578125, 0.35595703125, -0.1607666015625, -1.009765625, 1.1650390625, -1.33203125, 0.8818359375, -0.85009765625, -0.81591796875, -0.310791015625, 2.6328125, 1.1220703125, 1.3203125, -0.301025390625, -0.7744140625, -0.383056640625, -0.63232421875, 0.254150390625, 0.020172119140625, 0.29931640625, -0.52490234375, -0.447509765625, -0.348876953125, -0.830078125, 0.8935546875, -1.15625, 0.01255035400390625, -0.640625, -0.5732421875, -0.634765625, -1.412109375, -0.0858154296875, -0.0301666259765625, -1.025390625, -0.8759765625, -0.7802734375, 0.83642578125, 0.775390625, -0.2073974609375, -0.65625, -1.5927734375, 1.0078125, -0.2017822265625, 0.1578369140625, -0.472900390625, 0.0361328125, -0.1668701171875, 0.11456298828125, 0.138916015625, 0.17431640625, 0.68896484375, -0.64990234375, -0.21923828125, 0.58056640625, -0.5419921875, -0.057708740234375, -0.34912109375, -1.5830078125, -0.384033203125, -1.23828125, 0.71337890625, 0.5654296875, -0.88720703125, 1.3916015625, 0.09674072265625, -0.68603515625, -0.392333984375, 0.9619140625, -0.345458984375, -1.623046875, -0.3427734375, -1.466796875, -0.677734375, -0.65283203125, 0.032623291015625, -0.18603515625, 1.1484375, 0.0038776397705078125, -0.38720703125, 0.1807861328125, -1.90625, -1.552734375, 0.71533203125, 0.418212890625, -0.830078125, 0.900390625, -1.2109375, -0.09375, -1.4892578125, -0.025970458984375, -0.0433349609375, 0.5712890625, -0.03369140625, 1.736328125, -0.07196044921875, 0.59130859375, 0.21337890625, -1.076171875, -0.44140625, -0.5556640625, 0.28955078125, -0.650390625, -0.11053466796875, -0.6435546875, -0.9853515625, -2.392578125, 0.82763671875, 0.49658203125, -1.810546875, 1.5673828125, 2.150390625, -0.23779296875, -0.7138671875, 0.5673828125, -1.2802734375, 2.697265625, -0.0760498046875, 0.61572265625, -0.345458984375, -1.1552734375, 1.3037109375, 1.203125, -0.697265625, -0.9482421875, 0.26513671875, -0.8369140625, -2.369140625, 0.041412353515625, 0.5576171875, 0.020782470703125, 1.0, -1.2783203125, 0.884765625, 0.1461181640625, 1.6328125, -0.53515625, 1.5537109375, 0.09808349609375, -0.89599609375, -0.9775390625, 0.6533203125, 1.9326171875, 0.483154296875, -0.11602783203125, -0.1260986328125, -1.1064453125, 1.1318359375, 0.55517578125, 0.720703125, -1.857421875, 0.06463623046875, -0.257080078125, 1.037109375, -0.95947265625, -0.2432861328125, 0.6318359375, -0.85107421875, 1.181640625, 1.583984375, 0.15087890625, 0.4033203125, 1.953125, 1.2880859375, -0.88232421875, 0.56396484375, 2.11328125, -0.69873046875, 0.1956787109375, 0.03302001953125, -1.259765625, -0.6044921875, -0.344970703125, -1.0869140625, -2.25390625, 1.5888671875, 0.68896484375, 0.3671875, -0.266845703125, 0.1463623046875, 0.11578369140625, -0.65869140625, -0.09619140625, -1.158203125, 0.33447265625, 2.126953125, -0.94970703125, -0.9482421875, 0.49365234375, 0.61328125, 0.347412109375, 0.49658203125, -1.07421875, -0.1514892578125, 1.421875, -0.39013671875, -1.43359375, -0.703125, -1.0517578125, -0.55224609375, -0.93408203125, -0.480712890625, 0.79248046875, -0.037750244140625, 0.50146484375, 0.21142578125, -0.2166748046875, -1.1279296875, -0.493896484375, -0.1602783203125, -0.53564453125, -0.54931640625, -0.1466064453125, -0.52001953125, -1.3173828125, 1.2978515625, 0.54248046875, 0.219482421875, 0.267578125, 0.30224609375, -2.109375, 0.865234375, -0.21923828125, -0.43994140625, 1.32421875, -0.798828125, 0.11627197265625, -0.256103515625, 0.54443359375, -0.8447265625, 1.6640625, 0.60107421875, -0.84619140625, -0.406005859375, 0.004547119140625, 0.748046875, 0.313720703125, -0.90673828125, -1.1416015625, -0.427978515625, -0.9638671875, 0.64599609375, -0.1781005859375, 0.5234375, 0.26171875, -0.67822265625, 0.0750732421875, -1.5986328125, -0.748046875, -0.9755859375, 1.0673828125, -2.09765625, 0.8603515625, -0.6064453125, 0.00742340087890625, -1.6494140625, -2.1484375, -0.71240234375, 0.1373291015625, 0.2861328125, -0.06781005859375, 1.1318359375, -1.1142578125, 1.1484375, 0.98193359375, 1.4541015625, 0.039703369140625, 0.82421875, -1.1376953125, 0.8125, 0.487060546875, 0.362548828125, -0.0806884765625, -0.36083984375, 1.1171875, -0.541015625, -0.01013946533203125, 0.2447509765625, 0.6455078125, -0.284423828125, -0.21240234375, 0.693359375, -0.40673828125, -1.171875, 1.2451171875, -0.63623046875, 0.3662109375, 0.415283203125, 0.04852294921875, 0.2626953125, -0.442626953125, -0.50048828125, 1.3876953125, -0.244140625, -1.25390625, 0.10516357421875, -0.630859375, 0.1868896484375, 0.5869140625, 0.4921875, 0.316162109375, 0.849609375, -1.841796875, 0.0003371238708496094, -0.80224609375, 0.728515625, -0.272705078125, 0.78515625, 1.0546875, 1.412109375, -1.060546875, 0.2469482421875, 0.25244140625, 0.2021484375, 0.07281494140625, 0.37451171875, -0.96826171875, -0.290771484375, 0.498046875, 0.309326171875, 1.4189453125, -0.5400390625, -1.5634765625, 0.3837890625, 1.1298828125, -0.1475830078125, -1.1689453125, -0.07275390625, 0.966796875, -1.0107421875, 0.8662109375, -1.1884765625, -0.035003662109375, -0.1766357421875, -1.833984375, -0.1065673828125, 0.703125, -1.25390625, 1.6904296875, 0.70751953125, 0.314208984375, 1.25390625, -1.025390625, 0.300537109375, -0.396728515625, 1.4375, 1.06640625, -1.703125, 0.0279998779296875, 1.193359375, 0.7255859375, 0.33837890625, -0.595703125, -0.0298004150390625, -0.67333984375, 1.22265625, 0.556640625, 0.84423828125, 0.265380859375, -0.607421875, 1.419921875, -0.62158203125, -0.28125, 0.0112457275390625, 1.5966796875, 0.67431640625, 1.267578125, -0.259765625, -0.57275390625, -0.298583984375, 0.2568359375, -0.428955078125, -0.1854248046875, 0.4716796875, 0.916015625, 2.62109375, 0.340576171875, -0.157470703125, 1.0341796875, -0.119873046875, 1.123046875, -1.1787109375, -0.2054443359375, 0.88427734375, 0.6748046875, -0.41845703125, 0.91943359375, 0.71142578125, 0.259521484375, -0.96044921875, -0.62255859375, 0.5830078125, 1.1181640625, 2.671875, -0.34765625, -1.25390625, 0.7421875, -0.47119140625, 0.607421875, 1.8037109375, 1.134765625, 0.350830078125, -0.86474609375, -0.52490234375, 0.072998046875, 0.50146484375, -0.7431640625, -0.434814453125, 1.1796875, 1.5126953125, -0.9951171875, 0.4365234375, 0.1282958984375, -1.4970703125, -1.267578125, 0.176513671875, 1.5029296875, -2.763671875, 0.234619140625, -0.400146484375, 1.6337890625, -1.236328125, 0.80224609375, 0.39404296875, -0.84716796875, 0.07025146484375, -0.332275390625, 0.1419677734375, 0.64208984375, -0.357666015625, 1.115234375, -1.2373046875, 0.104736328125, -0.1253662109375, -0.77734375, 0.5986328125, 1.3642578125, 0.1346435546875, 1.1650390625, -0.7763671875, -0.900390625, 0.58203125, 0.7900390625, -2.4375, -1.859375, 1.0693359375, -0.6767578125, -0.4365234375, -0.66845703125, 1.0263671875, -1.517578125, -0.2313232421875, 0.73046875, -0.85400390625, -0.7705078125, -0.329833984375, -0.3212890625, -0.2218017578125, -0.0888671875, 0.429443359375, 1.5283203125, 0.1865234375, 1.6748046875, -0.95263671875, 1.25, 0.2298583984375, 0.453125, -0.3408203125, 0.7685546875, 0.92529296875, 0.125244140625, 0.369873046875, -0.370849609375, -0.212158203125, 0.970703125, -0.7060546875, -0.3681640625, 1.150390625, -1.443359375, -1.390625, -1.3349609375, -0.410400390625, -0.46044921875, -1.4130859375, 0.705078125, 0.33642578125, -0.135009765625, 0.263671875, 0.80859375, -0.0009632110595703125, -0.939453125, -0.505859375, 0.72119140625, -0.01837158203125, -0.30029296875, 0.37060546875, -0.222412109375, -1.0830078125, -0.9560546875, 0.5009765625, 0.705078125, -1.8408203125, -1.4375, -0.73828125, 2.40234375, 0.427490234375, -2.416015625, 0.54345703125, 1.669921875, -0.97412109375, -0.5322265625, 0.53369140625, 0.422119140625, -0.08319091796875, -1.3896484375, -0.7353515625, -0.56396484375, -0.99755859375, 1.533203125, 1.2373046875, -1.0029296875, -1.4453125, 0.73583984375, 2.828125, -0.95654296875, 0.93310546875, 1.8173828125, -0.701171875, -0.28369140625, -0.16357421875, 1.0595703125, -0.300537109375, -1.4638671875, 1.6044921875, -0.292236328125, 0.55029296875, 1.6376953125, -0.68798828125, -0.5068359375, -1.4208984375, 2.142578125, 0.58544921875, 0.65625, 0.91259765625, -0.99365234375, -0.298828125, -1.369140625, 1.2529296875, -0.556640625, 1.5400390625, 0.92333984375, -0.499267578125, 0.176513671875, 0.72314453125, -0.72998046875, -1.1962890625, -0.8193359375, 0.7041015625, -0.11212158203125, 0.1549072265625, -1.5322265625, -0.28515625, 1.00390625, -0.060943603515625, -1.8564453125, 0.84619140625, -1.052734375, -0.132568359375, -1.53125, -0.591796875, -1.7392578125, 2.302734375, 1.0693359375, 0.416259765625, -0.76416015625, -0.509765625, 0.432861328125, 0.63134765625, 2.2734375, -0.56884765625, 0.1217041015625, 1.0703125, 1.85546875, 2.484375, 2.046875, 0.42138671875, 0.74072265625, -0.6171875, 1.1591796875, 0.0712890625, -2.130859375, -0.45654296875, 0.051177978515625, -1.2861328125, -0.072509765625, -1.0341796875, 1.4296875, -0.1124267578125, -0.397705078125, 0.148681640625, 0.41552734375, -0.65771484375, -0.0008058547973632812, -1.173828125, 0.13720703125, -0.69189453125, -0.666015625, 1.10546875, 2.140625, 1.5439453125, 1.0458984375, 0.26513671875, 1.333984375, 0.30615234375, -0.317138671875, 1.6875, -0.67138671875, -1.3466796875, -0.50048828125, -1.138671875, -0.55810546875, -0.451416015625, 1.6044921875, 1.06640625, -2.052734375, -0.435791015625, -0.455078125, -0.29541015625, 0.41064453125, -2.0390625, 1.826171875, 0.751953125, 0.1507568359375, -1.9814453125, -0.29345703125, -0.469482421875, -1.765625, 0.180419921875, 0.68701171875, 0.94775390625, 0.276123046875, 0.65478515625, -0.087646484375, 0.12115478515625, 0.2252197265625, -0.666015625, 0.061614990234375, 0.2169189453125, 0.1336669921875, -0.2401123046875, 1.224609375, 0.61669921875, 0.297607421875, -0.47607421875, 0.0985107421875, -1.615234375, 0.23583984375, 0.974609375, 0.3056640625, 0.76953125, -0.7724609375, 1.0078125, -0.2587890625, -0.36767578125, 0.8701171875, -0.53173828125, -1.1484375, -1.283203125, -0.646484375, 0.29638671875, 0.9951171875, -0.73291015625, -0.56884765625, 2.2109375, 0.25341796875, 0.2080078125, 0.95849609375, 0.86572265625, -0.65283203125, 0.54638671875, 1.029296875, 0.87548828125, 0.77099609375, -0.339599609375, 0.489990234375, -1.4619140625, 0.334228515625, -1.2529296875, 0.00675201416015625, 0.2685546875, -0.404052734375, 1.46875, -0.420166015625, -0.049163818359375, 0.150634765625, 0.169677734375, -0.513671875, 1.017578125, -2.65625, -0.07403564453125, -0.95068359375, -0.489013671875, 2.283203125, 0.95947265625, 0.415771484375, 1.8935546875, 0.55078125, 0.22705078125, -0.6884765625, -0.253173828125, -1.2705078125, -1.6298828125, 0.951171875, -0.62060546875, -0.147216796875, 1.0556640625, -2.517578125, -1.197265625, -0.9951171875, 1.896484375, 0.53125, -0.439453125, -1.060546875, -0.97314453125, 1.1455078125, -1.650390625, 0.8095703125, -0.91943359375, -0.8046875, -0.78173828125, 0.0003933906555175781, 1.54296875, 1.076171875, 1.9873046875, 1.0244140625, -1.30859375, 0.8583984375, -1.0244140625, -0.120361328125, -0.1260986328125, -1.3681640625, -0.75732421875, 2.216796875, -1.0703125, -1.0595703125, 0.0232391357421875, -0.13916015625, 2.55078125, 0.2388916015625, 1.6640625, -0.98779296875, -1.3837890625, 0.90185546875, 0.274169921875, 0.66015625, 0.79052734375, -0.2315673828125, 1.2744140625, -0.376953125, 0.150390625, 0.0849609375, 0.65966796875, 1.140625, -0.346923828125, -1.5927734375, -0.73876953125, -0.079345703125, -1.0048828125, -2.181640625, -1.197265625, -2.53125, 1.2421875, -1.3203125, -0.23388671875, -0.305419921875, -1.3955078125, -0.71240234375, 0.76416015625, 0.958984375, -0.04132080078125, -2.025390625, 1.123046875, -1.0556640625, 0.7666015625, -1.455078125, 2.470703125, -0.6748046875, 1.4873046875, -0.60546875, -1.0908203125, 1.3095703125, -1.4453125, 0.59521484375, 0.607421875, -0.0654296875, -0.369140625, -1.4599609375, 0.40087890625, 2.4375, -0.80224609375, 1.232421875, -0.56640625, 1.0009765625, 0.177734375, 0.2215576171875, 0.434326171875, 2.568359375, 0.7431640625, 0.50390625, 1.134765625, 0.5791015625, 0.85009765625, 0.044586181640625, 1.0185546875, -0.099609375, -0.499755859375, -1.5615234375, 0.436279296875, 1.044921875, -0.426025390625, 0.0635986328125, 1.6337890625, -1.015625, 1.88671875, 0.900390625, 0.26171875, -0.366455078125, 0.716796875, -0.2119140625, -1.048828125, -1.4423828125, 2.529296875, -0.40380859375, 0.9013671875, -0.91796875, 0.49951171875, 0.83837890625, 0.9892578125, 0.2548828125, -1.0693359375, 0.50439453125, 0.2130126953125, -0.61669921875, -2.181640625, -0.189697265625, 0.039459228515625, -0.10272216796875, -0.04638671875, 1.6591796875, -1.3447265625, 1.216796875, -0.51123046875, -1.37890625, -0.4482421875, 0.81494140625, 0.6513671875, 0.08770751953125, 1.14453125, -0.30859375, -1.8740234375, -0.4677734375, -0.55615234375, -0.1656494140625, 1.185546875, -1.12109375, -2.015625, -1.388671875, 0.041717529296875, -0.412109375, 1.3017578125, -0.81689453125, -0.50146484375, 0.0236053466796875, 0.09771728515625, 0.46728515625, 1.3271484375, 2.314453125, 0.41259765625, -0.833984375, -0.529296875, -1.669921875, -1.4033203125, -1.021484375, 0.6474609375, 1.1943359375, -1.1318359375, 0.491455078125, -0.297119140625, 2.037109375, 0.71826171875, -0.798828125, -2.271484375, 0.27001953125, 1.333984375, 0.44921875, 1.5849609375, 1.35546875, -1.8388671875, -0.1829833984375, -0.232666015625, -0.77392578125, -1.8486328125, 0.86376953125, 2.92578125, 0.08673095703125, -0.027557373046875, -0.9169921875, -0.66552734375, -1.15234375, -0.4169921875, 0.56640625, -0.58740234375, 1.1005859375, -0.2149658203125, 1.751953125, 1.0546875, -1.013671875, -0.1251220703125, -1.2158203125, 0.89453125, 0.060577392578125, 0.32177734375, 0.63037109375, 0.3408203125, 0.373046875, -2.5, 1.5439453125, 0.70458984375, 1.3564453125, 0.3486328125, 0.62890625, 0.5498046875, 1.4033203125, -0.59765625, 0.1627197265625, 0.85498046875, 0.83642578125, 2.474609375, 1.2646484375, 0.4609375, 0.09332275390625, 0.3984375, 0.85986328125, -0.1212158203125, -0.60009765625, -0.0794677734375, 1.0107421875, -0.2213134765625, 0.49267578125, 0.05487060546875, 0.62939453125, -2.705078125, 0.29931640625, -0.59814453125, 0.63623046875, -0.91748046875, -0.818359375, 0.39599609375, -0.439453125, 2.501953125, 1.4638671875, 0.234130859375, -0.4521484375, 0.54150390625, 3.23046875, -0.1343994140625, -0.0213623046875, 0.3828125, 1.25390625, -0.43408203125, 0.1260986328125, 1.025390625, 0.677734375, -1.4091796875, 0.31103515625, 0.4580078125, 0.5185546875, -1.376953125, -0.119140625, 0.03167724609375, -1.1826171875, 1.630859375, 0.287841796875, -2.2890625, 1.080078125, -0.0936279296875, 1.2578125, 0.84765625, 1.25390625, -1.181640625, 0.283203125, -0.55859375, -0.06927490234375, -0.308837890625, 2.0546875, -1.32421875, 0.7705078125, 0.83447265625, 1.197265625, 0.93505859375, 0.1241455078125, 0.30078125, 1.068359375, 1.623046875, -0.68359375, -0.81689453125, 2.193359375, -0.0931396484375, -0.1982421875, 0.0830078125, -0.1937255859375, 0.27978515625, -0.119873046875, 0.00432586669921875, -0.193603515625, -0.75830078125, -1.1630859375, 0.2010498046875, -0.9912109375, 0.09967041015625, 0.80908203125, -1.4306640625, 0.4140625, 0.121826171875, 0.1605224609375, -1.671875, 1.7890625, 0.1497802734375, -0.0004088878631591797, 1.361328125, -0.09637451171875, 0.9072265625, 0.08392333984375, -0.84912109375, -0.515625, 0.93994140625, 0.77587890625, 1.326171875, 1.197265625, -0.0380859375, 0.40380859375, 0.64990234375, -0.8720703125, -1.1064453125, -0.28662109375, 0.00804901123046875, -1.078125, 1.263671875, -0.0858154296875, 0.59130859375, -0.50830078125, -0.20556640625, -0.68798828125, -1.2197265625, 1.7294921875, -1.3310546875, -0.49560546875, 1.294921875, -0.468505859375, 0.158203125, 1.94921875, 1.203125, -0.3798828125, 0.0134124755859375, -0.28515625, 0.30224609375, 0.273193359375, 0.7529296875, 0.64208984375, 0.67236328125, 2.005859375, -0.59326171875, 0.65966796875, 0.1953125, 1.40625, -0.406494140625, -1.2734375, -0.438232421875, 0.65771484375, -0.6201171875, -0.117431640625, -2.1640625, 1.099609375, 0.155029296875, -0.058441162109375, -0.994140625, -2.158203125, -0.310302734375, 0.354736328125, -0.337646484375, 0.7236328125, -0.29833984375, 1.3505859375, -0.658203125, 1.078125, 1.43359375, -0.370849609375, -0.830078125, 0.34716796875, -0.31298828125, -0.68994140625, -1.8173828125, -0.04693603515625, 1.361328125, -1.41796875, 0.537109375, 0.1826171875, 0.0191192626953125, -0.99169921875, 1.279296875, 1.228515625, 0.1258544921875, -0.61083984375, -0.0816650390625, 0.06292724609375, -0.74755859375, -1.826171875, -0.544921875, -1.529296875, 0.81005859375, -1.4111328125, 0.1292724609375, -0.421142578125, -0.3251953125, 1.4462890625, 0.51904296875, -0.50830078125, -1.64453125, 0.8115234375, 0.2388916015625, 0.0160369873046875, -0.59033203125, 0.646484375, -0.9736328125, 1.0634765625, -1.5859375, 0.95556640625, -0.219970703125, -0.08184814453125, 2.20703125, -0.364013671875, -0.250244140625, 0.73291015625, 0.2005615234375, -1.189453125, 0.9638671875, -1.4931640625, 0.055511474609375, -0.8623046875, -2.013671875, 0.6259765625, -1.8984375, 0.10205078125, 0.314697265625, -0.50732421875, -0.73193359375, -0.29736328125, 1.5341796875, -1.646484375, -1.8203125, -0.02020263671875, 0.7197265625, 1.81640625, -0.37939453125, -0.6904296875, 1.2109375, 0.230224609375, -0.0472412109375, -0.08782958984375, 1.4990234375, 1.8203125, -0.70947265625, -1.7666015625, -0.51953125, 0.44580078125, 0.258544921875, -0.4140625, 0.304443359375, -0.8212890625, 0.2061767578125, 1.03515625, 2.619140625, 1.640625, 0.9072265625, 0.160888671875, -2.076171875, -2.505859375, 0.2095947265625, -3.146484375, -0.0333251953125, 0.486083984375, 0.59326171875, 1.421875, -0.11334228515625, -0.8916015625, -0.40771484375, -1.21875, -0.0198822021484375, 0.354248046875, 0.288818359375, 0.2178955078125, -0.59130859375, 0.08758544921875, -0.314208984375, -0.6416015625, -0.70458984375, 1.2451171875, -1.1640625, 0.9072265625, 0.180419921875, -0.43896484375, -0.68017578125, -0.132080078125, 1.3740234375, 1.0498046875, -0.54296875, -1.1904296875, 0.892578125, -1.1943359375, -1.021484375, 0.429931640625, -1.3076171875, -0.288818359375, 0.70849609375, 1.0517578125, -0.9736328125, -1.6220703125, 2.24609375, 0.6298828125, 0.8037109375, -1.259765625, 0.8447265625, 0.52978515625, 0.365478515625, 0.499267578125, -0.67822265625, 0.2144775390625, -0.1180419921875, 0.23486328125, -0.5849609375, 1.04296875, -1.1533203125, 0.35791015625, 1.8955078125, -1.490234375, 0.7041015625, 1.044921875, 0.03814697265625, 1.1220703125, -0.87353515625, 0.57861328125, -0.7578125, -0.368896484375, 0.19189453125, -1.1064453125, 0.125, -1.0185546875, 1.16015625, 1.501953125, -2.201171875, 0.970703125, -0.50927734375, -0.91748046875, 1.5869140625, -0.07867431640625, 0.410888671875, 2.287109375, 0.60888671875, 0.09393310546875, -1.669921875, 0.206787109375, -0.71826171875, 0.93359375, -1.15625, 2.10546875, 0.9501953125, -0.251953125, -0.74853515625, -1.1904296875, -1.806640625, 1.5947265625, 0.80810546875, -0.359375, 0.71142578125, -0.037322998046875, 1.09375, -2.263671875, 0.04083251953125, 0.6552734375, 0.488525390625, 0.1124267578125, -0.4912109375, -0.525390625, 0.68994140625, -1.2734375, -0.673828125, 1.2958984375, 1.7880859375, 0.787109375, 0.88916015625, -0.0745849609375, -1.1328125, 0.1842041015625, -1.7529296875, -0.355712890625, -0.425048828125, -0.67041015625, -0.87548828125, -0.498291015625, 0.595703125, 0.6494140625, -0.83251953125, -0.461181640625, 0.205078125, -0.487060546875, 0.701171875, 1.078125, -1.0078125, -0.0265045166015625, -0.38427734375, -0.9130859375, 0.6591796875, 0.067626953125, -0.260986328125, -0.414306640625, -0.01155853271484375, 0.22314453125, -1.4599609375, 0.032379150390625, 0.90869140625, -1.0439453125, 1.134765625, 0.290283203125, 1.052734375, -0.93896484375, 0.8427734375, 0.37890625, 0.185302734375, -2.0703125, -0.81884765625, 0.0821533203125, -1.001953125, 0.2088623046875, -0.1376953125, 0.43408203125, -0.036346435546875, 0.80126953125, 1.4228515625, 1.4619140625, 1.3583984375, 0.0304412841796875, -0.95361328125, -0.249755859375, 0.9326171875, -0.0611572265625, 0.81103515625, -0.328125, 0.74951171875, 0.04913330078125, -0.529296875, 0.429443359375, -1.41796875, 0.1868896484375, -0.51318359375, 0.15673828125, -0.276123046875, 0.619140625, 0.208984375, 1.3623046875, -1.04296875, 0.5478515625, 1.1787109375, -1.3837890625, -0.47119140625, -1.703125, -0.1632080078125, 0.38623046875, -0.55078125, 0.476806640625, -1.205078125, 0.537109375, -0.91552734375, 0.455810546875, 1.255859375, 1.328125, -1.4091796875, -0.8857421875, 0.0108184814453125, 0.50390625, 0.7294921875, 0.09039306640625, -1.70703125, 0.191650390625, 2.111328125, -0.5205078125, -1.18359375, -0.3037109375, 0.98193359375, -0.88330078125, 1.4912109375, -1.4296875, 0.34619140625, -1.109375, -0.73583984375, 0.3408203125, 0.46044921875, -0.043792724609375, -1.1201171875, 0.273681640625, 1.0595703125, -0.19921875, 0.71875, 2.14453125, 0.95654296875, 0.7587890625, 0.2724609375, 0.1533203125, -0.0052947998046875, -1.2744140625, 0.70068359375, 0.07525634765625, 0.08831787109375, -0.7314453125, -1.1259765625, 0.1412353515625, -1.03515625, -0.72119140625, 0.10919189453125, -0.50537109375, 0.662109375, 1.224609375, -1.34375, 1.28515625, -0.388671875, -0.41259765625, 0.174072265625, -0.471435546875, 0.70458984375, -1.5302734375, 0.16015625, -0.2890625, -1.0166015625, -0.10870361328125, 0.7138671875, -0.02294921875, 1.65234375, 1.24609375, -0.8623046875, 0.05487060546875, -0.6279296875, 0.953125, -0.2783203125, -1.642578125, -0.039794921875, 1.8037109375, -0.67333984375, -0.91845703125, -1.18359375, 2.84765625, -1.4599609375, -0.1502685546875, -0.70751953125, 0.837890625, -0.388427734375, -1.3193359375, -0.7783203125, 0.388427734375, -0.1328125, 0.2138671875, -0.25, 0.365234375, 2.322265625, -0.712890625, 0.48876953125, -1.1328125, -0.841796875, 3.482421875, 0.89453125, 0.8955078125, 0.982421875, 0.4501953125, -1.056640625, -0.57275390625, -0.52587890625, 2.08984375, -0.69287109375, 1.71484375, 0.9189453125, 1.3203125, 0.60888671875, -1.59765625, 0.00409698486328125, -0.68505859375, -0.195556640625, -0.384033203125, -1.0615234375, 2.083984375, 0.1484375, 1.3427734375, -0.0102691650390625, 0.3623046875, 1.076171875, 0.0625, -1.0751953125, -0.384765625, 0.36181640625, -0.447998046875, -0.222900390625, 0.6484375, -0.92578125, -1.6943359375, -2.033203125, -1.486328125, -1.611328125, -0.89208984375, -1.736328125, 0.2498779296875, -0.410400390625, 0.30419921875, -0.38330078125, 0.2052001953125, -0.1368408203125, 0.284423828125, -0.2529296875, -1.88671875, -0.053863525390625, -1.755859375, -0.280517578125, 0.80712890625, 0.4541015625, 0.3564453125, -0.07049560546875, -0.70556640625, -0.051300048828125, 0.9677734375, 2.224609375, -1.66796875, 1.7861328125, 1.9208984375, 0.274169921875, -1.78515625, -0.85498046875, 0.4775390625, 0.427490234375, -0.0740966796875, -0.83740234375, -0.436767578125, 0.30224609375, 0.705078125, 0.94921875, 0.1412353515625, 0.341552734375, 1.7314453125, 2.0234375, 1.701171875, -1.60546875, 1.5048828125, -1.1435546875, -0.90869140625, -0.66162109375, -0.521484375, 0.94677734375, -0.5673828125, 1.162109375, -1.16796875, 0.2802734375, -1.24609375, 2.3515625, 0.6826171875, -2.29296875, -0.89794921875, -0.43408203125, 0.95751953125, 0.837890625, -1.8212890625, 0.451416015625, -0.96337890625, 1.2451171875, -0.453125, -1.478515625, 2.1796875, -0.00479888916015625, -1.6533203125, 0.2122802734375, -1.8798828125, 1.3056640625, 0.98583984375, -0.061004638671875, 0.24951171875, -1.25390625, 0.74365234375, -1.0576171875, 0.399169921875, 0.517578125, -0.283935546875, 0.1337890625, 0.1781005859375, -2.041015625, 0.058685302734375, -0.6005859375, 1.0869140625, 1.1083984375, -0.1485595703125, -1.3408203125, -1.08984375, 0.87939453125, -0.52978515625, -1.4775390625, 0.489501953125, 0.031524658203125, -1.1611328125, 1.08203125, 1.3427734375, 0.12103271484375, -0.1885986328125, 0.9501953125, 0.133544921875, 1.1767578125, 1.140625, 0.71923828125, 0.54345703125, 0.03729248046875, -0.59765625, -0.158203125, 0.52294921875, 0.720703125, 0.97216796875, -0.33837890625, -2.115234375, 1.6689453125, -0.2391357421875, -1.9423828125, -1.1982421875, -0.19287109375, -0.06280517578125, -0.72216796875, -1.3486328125, 0.79248046875, -0.0321044921875, 0.76513671875, -0.5166015625, 1.337890625, -0.049072265625, -0.5205078125, 2.34765625, 0.496826171875, -1.0771484375, 0.66943359375, -0.12457275390625, -0.41162109375, 0.423583984375, 0.587890625, -0.52099609375, 0.748046875, 0.1334228515625, -0.448974609375, 0.7421875, 0.01042938232421875, 1.5126953125, -1.40625, 1.6826171875, -1.5810546875, -0.0772705078125, 1.966796875, 1.443359375, -0.10821533203125, -0.798828125, 0.1484375, 0.93896484375, 1.25390625, 0.30517578125, 0.5517578125, 1.109375, 1.9267578125, 0.93359375, 1.18359375, 2.18359375, -0.68115234375, -2.0546875, -0.071044921875, -0.3056640625, 0.2447509765625, 0.04107666015625, -0.9814453125, 0.478271484375, -1.2392578125, 0.11279296875, 1.107421875, -0.195068359375, -0.12445068359375, -0.060943603515625, 0.06573486328125, 0.6328125, -0.1363525390625, 1.072265625, 1.0263671875, 0.6337890625, -0.97509765625, -1.2197265625, -1.4052734375, -0.454833984375, 0.391845703125, 0.8984375, -1.10546875, -1.0908203125, 0.87109375, 0.51904296875, -1.8291015625, 0.03448486328125, -0.435791015625, 0.036163330078125, 0.30712890625, 0.259521484375, -1.701171875, 0.8896484375, -0.8505859375, -0.8017578125, -0.286376953125, 0.255615234375, 0.1871337890625, -0.8203125, -1.384765625, 0.498046875, 0.13134765625, 0.295166015625, 1.568359375, -1.37109375, 1.0751953125, -0.01192474365234375, 0.007373809814453125, 0.88525390625, 0.2802734375, 0.7861328125, -0.17626953125, 1.279296875, 1.486328125, -0.50537109375, 0.404541015625, -0.285400390625, 0.358154296875, 0.71533203125, -1.5595703125, -0.12225341796875, -0.7255859375, 0.042724609375, -1.111328125, 0.10125732421875, -0.716796875, -1.9697265625, -0.1923828125, 1.69140625, -0.372314453125, -0.258056640625, 0.151123046875, 1.1220703125, 1.916015625, 0.288330078125, 1.8876953125, -1.021484375, -0.7900390625, -0.0183258056640625, 0.90625, 0.3740234375, 0.42333984375, 0.4970703125, -0.60498046875, -0.740234375, 0.05084228515625, -0.43701171875, -1.0556640625, 2.1171875, -0.71142578125, 0.345947265625, -0.59716796875, 1.8037109375, -1.1533203125, -1.015625, -0.68896484375, -0.86962890625, 0.52685546875, 0.86083984375, -0.580078125, 0.272216796875, 2.15625, -1.119140625, 0.22998046875, 0.4140625, -1.025390625, 0.77392578125, 1.08984375, -0.7607421875, -0.0780029296875, -1.3271484375, -0.79296875, -0.5400390625, -1.1845703125, 1.662109375, -0.006343841552734375, 2.248046875, -0.497802734375, -1.3701171875, 0.231201171875, 0.217041015625, -0.22509765625, -0.25439453125, -0.85791015625, 1.4912109375, 0.301513671875, -0.79736328125, -0.501953125, -0.05267333984375, 1.1298828125, 0.443359375, -1.115234375, -0.64013671875, -0.77001953125, -0.57177734375, 0.49609375, -1.8330078125, 1.9375, -0.2003173828125, 1.4560546875, 0.93505859375, 0.026763916015625, 0.09881591796875, -0.59033203125, 0.6767578125, -1.4267578125, 0.533203125, -0.059722900390625, 1.9326171875, -0.51318359375, -1.0322265625, -0.943359375, 0.998046875, -1.724609375, 0.63818359375, 0.1136474609375, -0.18798828125, -0.1795654296875, -0.8544921875, 1.5224609375, -1.60546875, -1.3408203125, -1.7998046875, -0.349609375, 1.2822265625, 0.08941650390625, -0.403076171875, -0.315185546875, -1.408203125, 0.5625, -2.220703125, 0.75439453125, 1.841796875, 0.5869140625, 1.205078125, -0.12200927734375, -1.935546875, -0.77099609375, 0.2403564453125, -0.1385498046875, -2.189453125, 0.313232421875, -1.3603515625, -1.3056640625, 1.068359375, 0.034149169921875, 0.56689453125, -0.033050537109375, 1.6943359375, -0.1737060546875, 0.198974609375, 0.712890625, 1.279296875, -0.02142333984375, 0.6533203125, 0.76416015625, 0.87890625, 0.187744140625, 1.8935546875, 0.4873046875, -0.154052734375, 0.67041015625, 1.298828125, 0.89892578125, -0.349609375, -0.452392578125, 0.80908203125, 0.8251953125, -0.79052734375, 0.07098388671875, -1.849609375, 1.0439453125, 0.55615234375, 1.0927734375, -1.265625, -1.51953125, 1.40234375, 0.337158203125, -0.171630859375, 1.5029296875, -1.3525390625, -1.3076171875, -0.269775390625, 0.689453125, 0.262451171875, -0.54541015625, -0.669921875, 0.6611328125, 0.4912109375, 0.189208984375, 1.2392578125, 2.1484375, 1.373046875, -0.411865234375, 0.720703125, 0.3662109375, -1.380859375, -0.433837890625, -0.05780029296875, -0.041778564453125, -0.76708984375, 1.845703125, -0.09088134765625, -0.380615234375, -0.6328125, 0.52197265625, 2.044921875, -0.6982421875, -0.016265869140625, -1.802734375, -1.35546875, 1.564453125, 1.0126953125, -0.394287109375, 0.383544921875, 0.7900390625, 0.6875, 1.5498046875, 1.107421875, -0.131103515625, 1.09765625, 1.8076171875, -1.0302734375, -0.27734375, -0.0709228515625, 1.1083984375, 0.401123046875, 0.466796875, 0.11578369140625, -0.68310546875, -0.395263671875, 1.5673828125, -0.85595703125, -1.576171875, 0.06634521484375, -0.994140625, -0.90234375, 0.37890625, -0.77099609375, 0.364013671875, 1.2177734375, 0.043609619140625, -0.53662109375, 0.50341796875, -0.89599609375, 1.5009765625, -0.00835418701171875, 0.275634765625, 0.0362548828125, -1.7978515625, 0.328125, 0.224609375, -1.4501953125, -0.357666015625, 0.421142578125, -0.0379638671875, 1.455078125, 0.30712890625, 2.2265625, -1.240234375, -0.088134765625, 0.28955078125, 1.419921875, 0.058624267578125, -0.013702392578125, -2.001953125, -0.6953125, -0.6826171875, 0.11614990234375, 0.173828125, 3.009765625, -0.986328125, 2.349609375, -0.70068359375, -1.0380859375, -0.375732421875, -0.031402587890625, 0.37109375, -0.77392578125, 2.017578125, -1.12109375, 1.0166015625, -0.96630859375, -1.0244140625, 0.3095703125, -1.5791015625, -0.3388671875, -0.31005859375, 0.396484375, -0.31689453125, -0.6005859375, 1.4755859375, 0.533203125, 0.08941650390625, 0.1043701171875, -0.149658203125, -0.9658203125, -0.74560546875, -0.065185546875, 0.49169921875, -1.1201171875, 0.58837890625, 0.417236328125, 0.759765625, -0.791015625, 0.6611328125, 0.53857421875, 0.95947265625, -0.96533203125, -3.041015625, 0.890625, -0.322509765625, 2.35546875, -0.055084228515625, -1.92578125, 0.51123046875, -0.184326171875, 1.61328125, -0.529296875, -0.3037109375, -1.154296875, -0.05499267578125, 1.2978515625, -2.029296875, -1.5625, -2.642578125, -1.591796875, 0.451416015625, 0.79931640625, -1.162109375, -1.5546875, 0.302001953125, -1.5458984375, 0.50927734375, -0.254638671875, 0.60205078125, -1.708984375, 1.0673828125, -0.95263671875, 1.23828125, 0.67529296875, 0.75830078125, -0.76171875, -1.609375, 0.41015625, -0.325927734375, 1.0126953125, -2.125, 2.45703125, 0.60546875, -1.4912109375, -0.77001953125, 1.40625, 0.266357421875, 0.74267578125, 0.62548828125, 1.087890625, 1.689453125, -0.1690673828125, 0.0516357421875, 2.060546875, -1.9296875, 0.8232421875, -0.2030029296875, -0.09039306640625, -0.226318359375, 0.50927734375, 1.255859375, 0.96533203125, -0.98828125, 2.4765625, 0.0147705078125, -0.29541015625, 0.2022705078125, 0.1153564453125, -0.94970703125, -0.6669921875, 1.05078125, 0.195068359375, -0.486083984375, 0.404052734375, 1.283203125, 0.1055908203125, 0.3525390625, 1.498046875, 0.533203125, 0.09259033203125, 0.427001953125, 0.2130126953125, 0.165283203125, 0.65234375, -1.140625, -0.052337646484375, 0.52978515625, 0.180908203125, -0.0127410888671875, 0.5712890625, 0.153076171875, 1.6787109375, -1.44140625, -1.3251953125, -0.0723876953125, -0.3779296875, 0.65234375, -0.77978515625, -0.02655029296875, 0.6796875, 1.62109375, 0.00823974609375, -0.7900390625, -0.4609375, -1.65625, 0.16943359375, 0.04718017578125, -0.79248046875, -2.55078125, -1.1767578125, -0.7919921875, 0.89208984375, -0.2451171875, 0.73876953125, -0.372314453125, 1.1669921875, -0.193603515625, -0.32421875, -0.1358642578125, -0.70751953125, 0.560546875, -0.8798828125, -1.087890625, -0.53369140625, 1.41015625, -0.8681640625, -1.021484375, 0.1240234375, 0.25439453125, 0.990234375, 0.50244140625, -0.1854248046875, -0.76953125, 0.3984375, 0.71240234375, 1.203125, 1.6103515625, 0.36474609375, 0.67822265625, 1.0244140625, -0.853515625, 1.6103515625, -0.2398681640625, 0.8701171875, 0.62451171875, 0.412353515625, -1.2734375, -0.849609375, 0.048980712890625, 1.037109375, 0.254638671875, 2.083984375, -0.1778564453125, 0.372802734375, -0.29345703125, 0.040679931640625, -1.1943359375, 0.5234375, -1.068359375, -0.2398681640625, 0.07098388671875, 0.1912841796875, 1.0185546875, -0.28857421875, 0.259521484375, -0.876953125, 0.173095703125, -0.119384765625, -0.030975341796875, 0.476318359375, 0.165771484375, -0.8720703125, -0.153564453125, -0.228515625, 0.2161865234375, 1.4755859375, -1.1806640625, 0.6455078125, -0.5908203125, -0.461181640625, 0.41650390625, 0.370849609375, 1.9140625, -0.142333984375, 0.385498046875, -0.97412109375, 0.4462890625, -0.58740234375, -0.341064453125, 0.1923828125, 0.447021484375, 0.82958984375, 1.6044921875, 0.44775390625, -0.7265625, -0.587890625, -0.77978515625, 0.6669921875, -1.0546875, 1.2568359375, 0.229736328125, -0.71435546875, -0.9228515625, 0.19140625, -0.404296875, -0.79638671875, -2.11328125, -0.302978515625, 1.0380859375, 0.16259765625, -0.53466796875, 0.724609375, 0.9755859375, 0.859375, 0.2861328125, 1.2080078125, 0.8935546875, -2.234375, -1.4189453125, -0.9189453125, -1.353515625, 2.19140625, 0.50341796875, 1.4072265625, -1.296875, -0.1654052734375, -0.32421875, -1.529296875, 0.0243682861328125, 2.935546875, -0.356689453125, -0.89453125, 0.354248046875, 0.59619140625, 0.78564453125, 0.92138671875, -0.0213623046875, 1.0859375, -0.669921875, 0.494873046875, -0.1591796875, -0.0828857421875, -0.39990234375, 0.85400390625, -1.1689453125, -0.06951904296875, 0.0655517578125, -0.51611328125, 0.050811767578125, -0.33984375, -1.017578125, -0.9765625, 0.0726318359375, 0.03497314453125, -0.195556640625, -1.224609375, 0.048980712890625, -0.492431640625, 0.32568359375, -1.0771484375, 0.421142578125, 1.9638671875, 0.4609375, -0.533203125, 1.1708984375, -1.0341796875, -0.5419921875, -0.020538330078125, -0.53857421875, 1.2158203125, 0.93408203125, -0.68994140625, 0.75048828125, -1.4755859375, 0.58251953125, 0.88037109375, 1.08203125, 1.1904296875, -1.640625, 0.316650390625, 1.08203125, -0.050933837890625, -0.5703125, 0.5927734375, -0.1929931640625, 1.380859375, -0.541015625, 0.205322265625, -0.242431640625, -0.7060546875, -0.505859375, -0.0296478271484375, -1.2255859375, -1.255859375, 0.33837890625, 0.966796875, 0.9462890625, 1.0654296875, 0.93701171875, -1.2958984375, -1.966796875, 0.227294921875, -0.39208984375, -0.1561279296875, 1.392578125, 0.491943359375, 2.32421875, 0.204833984375, -1.8623046875, -0.09722900390625, 0.484130859375, -0.2435302734375, -0.249267578125, -0.80712890625, -0.7080078125, 0.6455078125, -0.34814453125, 1.359375, 0.77392578125, -0.485595703125, -0.311279296875, -0.51708984375, 0.30712890625, 0.4453125, 0.312744140625, 1.267578125, 0.75341796875, -1.0712890625, -0.2122802734375, -0.1697998046875, -1.0546875, -0.328125, 0.495361328125, 0.237060546875, -0.348876953125, 0.53759765625, 0.355224609375, 0.66796875, -1.7119140625, 0.1868896484375, 0.8828125, 0.036895751953125, -1.09765625, 0.9267578125, 0.284423828125, -2.474609375, -1.521484375, 0.489501953125, -1.1552734375, -1.1689453125, -0.2044677734375, 0.5400390625, 0.8427734375, 0.33544921875, -1.8623046875, -0.67578125, -1.1181640625, -0.21875, -0.489990234375, -1.3134765625, -1.1005859375, 0.45068359375, 1.5224609375, -0.95751953125, 0.58740234375, -2.216796875, -0.73876953125, 0.451904296875, -1.2099609375, -0.7333984375, 0.250732421875, 0.0833740234375, 0.1959228515625, 0.49267578125, -0.98291015625, 0.200927734375, -0.07623291015625, -1.8134765625, -0.56103515625, 1.56640625, 0.435546875, -0.875, 1.0537109375, -0.70751953125, -1.9931640625, -0.626953125, 0.28564453125, -2.330078125, 0.9169921875, 0.2021484375, 0.869140625, 0.032135009765625, -0.98388671875, -0.495849609375, 0.86083984375, 1.5361328125, -1.1982421875, 1.7880859375, 2.064453125, 2.041015625, -0.82421875, 0.427490234375, 0.806640625, 0.4814453125, 0.339111328125, -0.8427734375, -0.6875, 0.3291015625, 1.1435546875, 0.39306640625, 0.75146484375, 0.4248046875, -1.1923828125, 0.5712890625, -2.330078125, -0.332275390625, -1.0810546875, 1.0322265625, -2.046875, -1.697265625, -0.412841796875, 1.970703125, 0.98193359375, 0.434326171875, -0.99560546875, -0.501953125, -1.07421875, -0.1484375, 0.498046875, 2.029296875, -1.55859375, -0.63134765625, -1.3466796875, -0.4599609375, -0.81396484375, 0.87939453125, -0.96337890625, -0.82177734375, -0.86474609375, -0.325439453125, -0.6201171875, 0.50830078125, -1.3955078125, -1.798828125, -0.2022705078125, -1.333984375, -1.5556640625, 0.86669921875, -0.7919921875, -0.3251953125, -0.44140625, -0.27685546875, 1.66015625, 0.30908203125, -1.57421875, -0.80126953125, -1.0791015625, 0.07611083984375, -0.408935546875, 0.84423828125, 0.24462890625, 1.1435546875, 0.91748046875, -0.378173828125, 0.46826171875, -0.5283203125, -0.431884765625, -1.01171875, 0.90869140625, -0.53515625, -1.0068359375, -1.9619140625, 0.513671875, 2.86328125, -0.431396484375, 0.4228515625, -0.87451171875, 0.2320556640625, 0.497314453125, -0.1695556640625, 1.736328125, -0.7568359375, -0.76318359375, 0.04254150390625, 0.1866455078125, 0.22509765625, -0.5556640625, -0.583984375, 0.382568359375, -1.435546875, 0.409912109375, -1.140625, 0.7880859375, 1.126953125, -0.364013671875, 1.0126953125, 0.9169921875, 0.56298828125, 0.546875, 0.62158203125, -1.81640625, 0.10205078125, -0.490234375, -1.4833984375, -1.283203125, 0.272705078125, -1.5849609375, -0.479736328125, 0.441162109375, 0.27490234375, 0.34619140625, 0.81396484375, 0.59765625, -0.1888427734375, 0.32958984375, 0.273193359375, 0.69189453125, -1.4248046875, -0.0227508544921875, 0.41748046875, -0.180419921875, 0.94091796875, 0.7744140625, -0.693359375, 0.329345703125, 1.4326171875, -0.787109375, -0.5615234375, -0.4892578125, -0.6337890625, 0.391357421875, -0.384033203125, -1.4775390625, -1.8095703125, 1.6572265625, 0.10931396484375, 0.75341796875, 0.053619384765625, 0.320068359375, 1.2119140625, 1.5791015625, 1.40234375, 0.51123046875, -0.09930419921875, -0.255126953125, -1.34765625, -0.9638671875, -1.4833984375, -0.369384765625, -0.272705078125, 0.1231689453125, -0.85693359375, 0.994140625, -1.1923828125, 0.68798828125, -1.0263671875, 0.331298828125, -0.2227783203125, 1.07421875, 0.8935546875, 0.3076171875, -0.3388671875, -0.1669921875, 0.87255859375, 0.6943359375, -0.1483154296875, -0.74609375, -0.994140625, 0.66015625, -0.061065673828125, 0.406494140625, -0.5166015625, -1.9365234375, 0.818359375, -0.0936279296875, 0.27197265625, 0.69677734375, 2.154296875, -0.452392578125, 0.84033203125, 0.22314453125, 0.195068359375, -0.9765625, 0.99658203125, -2.126953125, 0.43212890625, 0.17431640625, 2.359375, 0.1409912109375, -0.71923828125, 1.818359375, -1.9228515625, -1.4892578125, 0.73974609375, -0.358154296875, 1.5, 0.74267578125, 0.78955078125, 2.04296875, 0.29833984375, 0.4443359375, -0.366455078125, -0.179931640625, 0.263671875, 0.51318359375, 0.626953125, 1.8671875, 0.0643310546875, 0.81689453125, 1.9638671875, 0.75830078125, 0.689453125, 0.6416015625, 1.11328125, 0.1474609375, -1.3564453125, 0.203369140625, -0.312255859375, -0.45263671875, 0.50146484375, 1.26953125, 0.484375, -0.39794921875, -0.9951171875, 1.3115234375, 0.3173828125, 2.705078125, -0.62109375, -0.11993408203125, 0.798828125, -0.64013671875, 2.271484375, 0.8818359375, -1.7998046875, 0.060028076171875, 0.79833984375, -1.3173828125, -0.67822265625, -0.454345703125, -0.444091796875, -0.039886474609375, 1.4990234375, 0.11798095703125, 2.03125, 0.1954345703125, 0.7724609375, -3.216796875, 0.84619140625, -0.134521484375, -1.3408203125, 0.026123046875, 0.417724609375, 1.1640625, -0.392578125, -1.611328125, 0.72265625, 1.1103515625, -0.456298828125, -0.88330078125, 0.73583984375, 0.5185546875, 0.99951171875, -1.2841796875, 0.88134765625, 0.1239013671875, 1.0849609375, 0.9794921875, -1.119140625, 0.73583984375, 1.0439453125, -1.1982421875, -1.537109375, 0.27587890625, 0.209716796875, -1.03515625, 0.483154296875, -0.2744140625, 0.55078125, 0.9423828125, 0.260986328125, 1.974609375, -0.80322265625, -0.368896484375, -0.83203125, -0.55908203125, -0.37255859375, -0.431884765625, 0.96875, 0.114013671875, -0.405029296875, -1.64453125, 0.359130859375, -1.998046875, -0.22900390625, 1.8896484375, -1.03125, 0.18896484375, 0.10699462890625, 0.15576171875, 0.333251953125, -1.306640625, -1.455078125, -0.2578125, -1.330078125, -1.9501953125, -1.1259765625, -0.47802734375, -0.73583984375, 1.265625, 1.1025390625, -1.9775390625, 1.5869140625, -0.54541015625, -1.017578125, 0.2156982421875, 1.25, 0.74365234375, 0.380615234375, -1.00390625, 1.236328125, -0.42822265625, -0.2646484375, 0.48876953125, -0.491943359375, -0.6962890625, 0.257080078125, 0.144775390625, 1.064453125, -1.423828125, 0.154296875, -0.39306640625, 0.1292724609375, -2.42578125, 0.1923828125, -0.0010461807250976562, 0.234130859375, -0.459228515625, 0.9638671875, -0.666015625, 0.1988525390625, 0.0859375, -0.6640625, 0.08087158203125, -1.0947265625, -1.1533203125, 2.04296875, 0.4091796875, -0.49609375, 1.3935546875, 0.96630859375, 1.1552734375, -1.384765625, 0.6630859375, -0.79248046875, 0.441650390625, -1.294921875, 0.92236328125, -0.1712646484375, 0.88916015625, 0.9609375, -1.083984375, 0.256591796875, 0.89501953125, 1.517578125, -0.9130859375, 0.1231689453125, -0.90380859375, -1.1015625, 0.189208984375, -0.72900390625, 0.06597900390625, 0.1890869140625, -0.6943359375, 0.720703125, -1.177734375, -0.053985595703125, -0.267333984375, 2.63671875, -1.4326171875, 0.2880859375, -0.41162109375, -0.90283203125, 0.75, -1.85546875, 2.19140625, -0.28466796875, 1.1826171875, 0.334228515625, 0.2230224609375, 1.01953125, 0.29541015625, 0.269287109375, -0.60205078125, -0.33154296875, 0.045166015625, 0.57080078125, 0.2374267578125, 1.0166015625, -0.381591796875, 0.78125, -1.5078125, -0.1527099609375, -0.5126953125, -1.4091796875, -0.225341796875, -0.21875, -0.1358642578125, -0.7646484375, -0.11224365234375, 0.82373046875, -0.375244140625, 0.9287109375, -0.89697265625, 0.62744140625, 0.1588134765625, -0.63818359375, -0.6484375, -1.041015625, -1.021484375, 1.0400390625, 0.779296875, -0.8994140625, -1.0361328125, -0.043060302734375, 0.298828125, -1.03515625, 1.1474609375, -0.966796875, -1.2177734375, -0.467529296875, -1.0185546875, 0.9892578125, 0.09130859375, 0.54736328125, -1.9697265625, 0.51171875, 0.150390625, 0.416015625, -0.6005859375, 0.9755859375, -0.98974609375, 1.2548828125, -1.326171875, 0.5498046875, 0.32568359375, 1.1025390625, 1.2109375, -0.4833984375, 0.091064453125, -1.7666015625, -0.9443359375, -0.89697265625, 0.15869140625, 0.9423828125, 1.1875, 0.1602783203125, -0.149658203125, -0.09124755859375, -0.92919921875, -1.4296875, -0.1326904296875, -0.260009765625, -1.3310546875, 0.09033203125, 0.8818359375, 1.1806640625, 0.517578125, 0.308349609375, 1.333984375, -0.2469482421875, -1.6728515625, 0.51171875, -1.40234375, -0.27685546875, -1.9482421875, -2.435546875, -1.0263671875, -0.61328125, 0.022369384765625, 2.41015625, -1.5908203125, 0.037994384765625, 1.1328125, -0.448486328125, 0.354248046875, -1.1015625, 0.097412109375, 0.1160888671875, 0.46826171875, -0.402587890625, 0.1790771484375, -0.6220703125, 1.5126953125, 1.7890625, -1.0302734375, 0.1002197265625, -2.15625, -0.0195465087890625, -0.6533203125, -1.8837890625, 1.935546875, -1.810546875, -1.6962890625, -0.33251953125, -1.2919921875, 0.46484375, 0.367431640625, -0.72509765625, -0.5341796875, 1.2666015625, 0.042999267578125, -0.39892578125, 0.052337646484375, -1.18359375, 1.94140625, 0.260009765625, -0.56298828125, -1.1572265625, 1.3369140625, -0.0145263671875, 1.7763671875, -0.10919189453125, 2.2421875, -0.82421875, 0.6484375, -0.429443359375, 1.8427734375, 0.26123046875, 0.849609375, -0.3251953125, -1.451171875, -1.36328125, 0.60107421875, 0.251708984375, -0.2137451171875, 0.75, 1.5537109375, -0.41455078125, -1.361328125, -0.7958984375, -0.2276611328125, 1.0615234375, -0.369873046875, 0.4091796875, 1.34375, -0.62890625, -1.638671875, -0.02923583984375, -1.30078125, -0.0765380859375, -1.109375, -0.09503173828125, -0.07177734375, 0.560546875, 0.1370849609375, -0.11920166015625, 0.0992431640625, -0.1865234375, 1.060546875, -0.2017822265625, -0.20068359375, 0.4833984375, -2.533203125, 0.134033203125, -1.7880859375, 1.7109375, 0.83544921875, -0.2193603515625, -1.1875, -0.6279296875, -0.03369140625, -0.9189453125, -1.8076171875, 0.85888671875, 1.2294921875, -0.1798095703125, 0.265380859375, -0.60400390625, 0.369384765625, -0.84765625, 1.58984375, -1.3505859375, 1.234375, 2.2578125, 0.18896484375, 0.724609375, -1.9326171875, 1.9775390625, 1.263671875, 0.54443359375, 0.2259521484375, -0.3017578125, 0.6796875, 0.212646484375, -0.225341796875, -0.64111328125, 0.312744140625, -0.0816650390625, -0.1571044921875, -0.06744384765625, 0.9345703125, -0.51708984375, 0.1768798828125, 0.2088623046875, 1.9267578125, 0.81640625, 0.22021484375, -1.35546875, -0.712890625, 0.42919921875, 0.2763671875, 0.16259765625, 0.50146484375, -0.06976318359375, 0.359130859375, -1.876953125, -0.5380859375, 0.28564453125, -0.5712890625, 1.4296875, 0.44580078125, 1.0703125, 1.734375, -2.326171875, -0.10003662109375, 0.826171875, -0.358154296875, 0.82666015625, -1.556640625, -0.5537109375, 0.0693359375, -2.009765625, 0.304443359375, 0.70556640625, 0.266357421875, 0.51806640625, -0.10479736328125, -0.342041015625, 1.5478515625, -0.7998046875, 0.310302734375, 0.7470703125, 1.8203125, -0.4482421875, 0.302490234375, -0.81005859375, 1.669921875, -0.136474609375, -1.0068359375, 0.268310546875, 0.16162109375, -1.8623046875, -0.744140625, -0.3515625, -0.51171875, -0.1937255859375, -0.63330078125, -0.85791015625, -1.4599609375, -0.2093505859375, -0.378662109375, -0.58740234375, -2.00390625, 2.017578125, 0.1644287109375, -0.241943359375, -1.2705078125, -0.200927734375, 1.1416015625, -0.0006556510925292969, 1.1357421875, -0.026947021484375, -0.54150390625, -0.93701171875, -2.927734375, 1.232421875, -0.486572265625, 0.48486328125, -0.68994140625, 1.005859375, 0.0582275390625, -0.3681640625, -1.7548828125, -0.6494140625, 0.666015625, 0.2626953125, 1.3583984375, -0.0345458984375, -0.35205078125, 0.02362060546875, -0.861328125, -0.60400390625, 0.7978515625, 0.0838623046875, 0.6708984375, 0.87548828125, 0.74658203125, 0.525390625, -1.5498046875, -0.448486328125, 1.212890625, -0.390869140625, -1.541015625, -0.83203125, 0.3759765625, 0.308349609375, 0.82861328125, -0.1610107421875, 0.6142578125, -1.7265625, 0.802734375, -0.27294921875, 0.2244873046875, 1.3193359375, 0.6005859375, -2.048828125, -0.419921875, -1.3544921875, 0.304931640625, 0.471923828125, 0.708984375, 0.3779296875, 0.1767578125, -0.99853515625, -0.04803466796875, 0.2978515625, 0.221923828125, -1.8720703125, -0.272216796875, -0.34765625, -0.2320556640625, 0.68896484375, -1.271484375, 1.3046875, 1.4677734375, 0.110107421875, 0.70263671875, -1.109375, 0.58544921875, 0.0863037109375, -1.10546875, 0.11212158203125, -0.405029296875, -0.0562744140625, -1.12109375, -1.216796875, 0.12353515625, -0.148193359375, -0.8515625, 1.8671875, -0.88720703125, 0.35791015625, -1.74609375, 0.6650390625, 1.1533203125, -0.9033203125, -0.564453125, -1.634765625, 0.1243896484375, -0.359619140625, 0.94482421875, 1.0478515625, -0.1898193359375, -0.5634765625, -1.7470703125, -0.52978515625, 1.7890625, -0.84130859375, 0.390625, 0.61181640625, 1.3017578125, 1.150390625, -0.2266845703125, -0.41357421875, -0.67578125, 0.006687164306640625, -0.186767578125, 1.23046875, -0.490966796875, 0.6982421875, 1.5712890625, -0.70947265625, 0.1307373046875, 2.49609375, -0.4873046875, 0.4775390625, -0.64892578125, 1.3828125, 0.34814453125, 1.5654296875, 0.315673828125, -0.19091796875, -0.364990234375, -0.69970703125, 0.8837890625, 0.08148193359375, 0.36083984375, -0.5341796875, -0.58984375, 0.279541015625, -0.798828125, -0.82275390625, -0.465576171875, -0.51611328125, 0.31982421875, 1.1025390625, -0.136962890625, 0.5859375, 0.200927734375, 0.61669921875, 0.035400390625, 0.63720703125, -0.67138671875, 0.6875, -0.88037109375, -1.2724609375, 1.0302734375, 1.4296875, 0.07379150390625, -1.08203125, -0.00904083251953125, 0.68212890625, -1.150390625, 1.3779296875, -0.94677734375, -0.57275390625, -0.284912109375, 1.1044921875, 1.3447265625, 1.650390625, 0.07318115234375, 0.70751953125, -0.335693359375, 0.323974609375, -1.1689453125, 0.19921875, 0.6552734375, 0.98193359375, 1.576171875, -0.1134033203125, 0.14013671875, 1.0751953125, 0.205810546875, -1.376953125, 1.3017578125, 0.94482421875, 0.01050567626953125, 0.29541015625, 0.215576171875, 0.70849609375, 0.0142059326171875, -0.0211639404296875, -0.27587890625, 0.287841796875, 0.0192718505859375, -0.2301025390625, 0.1790771484375, -0.7470703125, -1.0263671875, -0.2666015625, 0.79296875, -1.0576171875, -2.24609375, -0.16796875, -1.0380859375, 0.1993408203125, -0.7578125, -1.7880859375, -0.0665283203125, 0.29736328125, -0.002471923828125, -0.619140625, 1.619140625, -0.9248046875, 0.58154296875, 0.25390625, -0.92919921875, -1.96875, 0.583984375, -0.44384765625, -0.8525390625, -0.48681640625, -0.302001953125, 0.139404296875, 1.6494140625, -0.876953125, -0.300048828125, 0.5478515625, 1.1240234375, 0.039093017578125, 0.3203125, 1.0107421875, 0.405517578125, 0.630859375, -1.6611328125, -0.86865234375, -1.4873046875, 0.387939453125, 1.80859375, -0.7392578125, -0.689453125, -0.396240234375, -0.06982421875, -2.298828125, 0.405517578125, 0.2142333984375, 0.285888671875, 0.4013671875, 0.022308349609375, 1.6376953125, -0.751953125, -0.9228515625, -0.49560546875, 1.2568359375, -0.42138671875, -1.3662109375, 0.399169921875, -1.7646484375, -0.1734619140625, -2.087890625, -0.2626953125, 0.5107421875, 0.1602783203125, 1.0322265625, 1.9921875, -0.7568359375, -0.1268310546875, -0.87548828125, 0.10858154296875, -0.1519775390625, -0.57080078125, 1.1982421875, -1.1201171875, -0.96728515625, -0.61669921875, -0.6123046875, 1.734375, -0.10186767578125, 0.2210693359375, 0.288330078125, 0.052032470703125, 0.1409912109375, 1.2841796875, 0.007625579833984375, 0.0269775390625, -1.2041015625, -1.1669921875, 1.099609375, -1.34375, 0.1141357421875, -0.7587890625, -1.4755859375, -0.9150390625, -0.51025390625, 0.35107421875, 0.137451171875, 1.7890625, -0.7919921875, -0.397216796875, -1.333984375, 1.310546875, 0.02093505859375, 0.2149658203125, -1.3125, 0.1455078125, 1.57421875, 0.454345703125, 1.0048828125, 1.5888671875, 0.7431640625, -0.28955078125, -0.771484375, 0.318359375, -0.65771484375, -0.19775390625, 1.1708984375, 0.67431640625, -0.2203369140625, 0.40234375, 1.7607421875, -0.9599609375, -1.3330078125, -0.92333984375, 1.7607421875, -2.16015625, -1.1357421875, 0.332763671875, -0.62451171875, 0.4248046875, -0.11541748046875, 0.84326171875, -0.426513671875, -0.712890625, 0.55126953125, -0.4072265625, -0.0965576171875, -1.5712890625, 0.48193359375, -1.5390625, 0.1578369140625, -0.51123046875, -0.6162109375, 0.388671875, -0.271484375, 0.1851806640625, 0.9140625, -0.2021484375, -0.358154296875, 1.009765625, -0.5048828125, -0.11041259765625, 0.73779296875, 0.262939453125, -1.8310546875, -0.3046875, -1.9140625, -1.140625, 0.46240234375, -0.2139892578125, 0.6865234375, -0.80859375, 0.1475830078125, 0.892578125, -0.5302734375, 0.669921875, 0.55615234375, 1.2255859375, 1.462890625, 1.77734375, -0.242919921875, 0.67236328125, -1.4521484375, -0.734375, -0.034454345703125, 0.432373046875, -1.56640625, 0.677734375, 1.66015625, -0.615234375, 0.650390625, -0.447265625, -0.7578125, 2.177734375, -3.03515625, 0.79833984375, -1.34765625, 0.09368896484375, 0.24853515625, 0.027923583984375, -0.322021484375, 0.6201171875, 3.203125, 0.376220703125, 0.5302734375, 1.35546875, 0.0369873046875, -1.09765625, 0.7568359375, 1.0947265625, -0.1556396484375, -0.2161865234375, -0.63525390625, -0.00783538818359375, -0.951171875, 0.58349609375, 0.058746337890625, 0.57080078125, 1.625, -0.196533203125, 0.055908203125, -0.83935546875, -2.298828125, 0.8505859375, 0.493896484375, 2.107421875, 0.8056640625, 1.099609375, 0.1629638671875, -1.0, 0.430419921875, -0.0736083984375, -0.33935546875, -0.3564453125, -0.515625, 1.0224609375, 0.33935546875, -1.33984375, -0.49853515625, -0.0220947265625, 1.798828125, -0.107666015625, -0.5234375, -1.7646484375, -0.2052001953125, -0.44384765625, -0.248046875, -1.52734375, 0.5791015625, -1.142578125, 1.1513671875, 1.044921875, -0.5341796875, 0.74267578125, -0.2012939453125, -0.5654296875, 0.69482421875, -0.0214996337890625, -0.415771484375, -0.51123046875, -1.890625, 0.90234375, -0.301513671875, -0.25146484375, -0.26953125, 0.666015625, -0.3203125, -0.1444091796875, -0.5068359375, 1.4326171875, -1.0498046875, 0.26611328125, 0.41064453125, -0.4013671875, 0.505859375, -0.2880859375, 0.666015625, 0.342041015625, -0.98291015625, -0.380126953125, 0.263671875, 0.08331298828125, -0.66259765625, 0.0004305839538574219, -1.076171875, 0.2037353515625, 0.52392578125, -0.326171875, 1.078125, 0.9755859375, -1.0322265625, -0.8330078125, -0.8310546875, -0.49267578125, -1.0859375, 0.0777587890625, 1.419921875, 1.103515625, -0.003337860107421875, 1.1181640625, -1.5654296875, 0.5625, 1.9365234375, 1.0029296875, 0.99267578125, -0.87841796875, 0.82568359375, 0.63720703125, -1.388671875, 0.61865234375, -0.2237548828125, 1.439453125, -0.64892578125, 0.88720703125, -1.4716796875, -0.409423828125, -0.448486328125, -0.479736328125, 0.91259765625, 0.11309814453125, -0.463623046875, -0.94482421875, -1.01171875, 0.54345703125, -1.30078125, -1.251953125, -0.35400390625, 1.6982421875, -0.16650390625, -0.1087646484375, -0.322998046875, 0.1744384765625, -1.2099609375, -0.798828125, -0.175537109375, 0.270751953125, 0.94140625, -0.261962890625, -2.884765625, 0.341796875, 0.6494140625, -0.10003662109375, -0.1142578125, -0.150634765625, -0.1817626953125, 0.8857421875, -0.9833984375, 0.298828125, -1.4365234375, 0.77392578125, -2.181640625, -0.365966796875, 0.128173828125, -1.58203125, -0.7275390625, 0.91845703125, 0.81982421875, 0.751953125, 0.435302734375, 1.333984375, 0.08184814453125, 0.53564453125, -1.2734375, 0.6826171875, -0.125, 0.3408203125, 0.81005859375, 0.5986328125, -1.7880859375, 0.830078125, -1.1455078125, 1.5390625, 0.410400390625, -0.080810546875, 0.58837890625, -2.3671875, 1.8984375, -2.6015625, -0.1741943359375, 0.312744140625, 0.54296875, -0.214599609375, 0.470947265625, -0.43310546875, 0.496337890625, -0.57373046875, -0.264892578125, 0.58837890625, -1.2578125, -0.95458984375, -0.01153564453125, -0.41943359375, -1.4111328125, -0.35009765625, 0.81201171875, -0.075439453125, -0.6005859375, -0.71875, 1.283203125, 0.232177734375, 0.481689453125, 1.6611328125, -0.459228515625, -0.865234375, -0.029876708984375, 1.515625, -2.435546875, -0.63427734375, -0.0357666015625, -0.029876708984375, 1.1279296875, -0.2113037109375, 0.5556640625, 0.89013671875, 1.4375, -0.90625, 2.59375, -0.77587890625, 1.021484375, -0.4814453125, 0.00592803955078125, 0.8837890625, 0.14013671875, 0.1090087890625, -0.136962890625, -0.572265625, -0.1317138671875, -1.1279296875, 0.5556640625, -0.130126953125, 1.7958984375, 1.00390625, -0.71533203125, 0.576171875, 0.02447509765625, 0.456298828125, -0.1383056640625, 0.362548828125, 1.197265625, -1.7275390625, 0.049102783203125, 0.75634765625, -0.40380859375, 1.08984375, -0.0452880859375, -1.150390625, -0.78564453125, 0.278076171875, 1.25, -1.8515625, 0.254638671875, 1.017578125, -0.5029296875, 0.34619140625, 0.341064453125, 0.238037109375, 2.333984375, 1.4970703125, 0.6904296875, -0.316162109375, -0.9111328125, 2.015625, -0.30224609375, 0.3828125, 0.765625, 0.87060546875, 2.302734375, -1.48828125, 0.371826171875, 2.513671875, 0.95947265625, -0.53759765625, 0.1341552734375, -0.0350341796875, 1.2900390625, -1.3671875, -0.83984375, 0.73486328125, 3.033203125, 0.9072265625, 0.09600830078125, 0.036285400390625, 0.3828125, 0.9052734375, -0.57080078125, -1.26953125, 0.76025390625, -2.044921875, -1.3017578125, -0.38720703125, 0.3291015625, 1.232421875, 0.07586669921875, 1.767578125, -0.61767578125, -0.279296875, -2.169921875, -0.241943359375, 0.826171875, 0.11749267578125, -1.1787109375, -0.35546875, -0.59130859375, 0.1278076171875, -0.26708984375, -0.100341796875, 0.250244140625, 0.398193359375, -0.69140625, -0.13916015625, -0.0623779296875, -2.19140625, 0.296875, -1.2392578125, 0.11163330078125, -1.396484375, -0.53662109375, -0.2225341796875, 0.96923828125, -0.7109375, 1.259765625, 1.970703125, -2.76953125, -1.2626953125, 0.29833984375, 0.364501953125, -0.892578125, 1.0859375, -0.006866455078125, -0.79638671875, 0.6435546875, 0.04815673828125, 0.333740234375, -0.68798828125, -2.263671875, 1.2763671875, 1.8701171875, 0.0288848876953125, 1.1533203125, -0.8310546875, -0.016693115234375, 1.4716796875, 1.8017578125, 0.7626953125, 1.4072265625, -0.2275390625, 1.337890625, -0.356201171875, 1.17578125, 0.3837890625, -0.2069091796875, 0.29833984375, 0.80908203125, 1.4052734375, 0.480224609375, -0.11236572265625, 0.4716796875, -0.07208251953125, 0.73193359375, -0.67236328125, -1.7919921875, -1.9775390625, 1.3515625, 0.49365234375, -0.81884765625, 0.73828125, 0.55029296875, 1.626953125, 2.00390625, -1.1904296875, 1.732421875, 0.9716796875, -0.9287109375, -0.027679443359375, -0.308837890625, -0.93115234375, -1.265625, -2.404296875, -1.673828125, 0.0943603515625, -0.78662109375, -0.7333984375, 0.88232421875, 0.30126953125, 0.2293701171875, 1.7568359375, 0.60791015625, -0.2342529296875, -0.1556396484375, -0.0736083984375, -0.03265380859375, 0.2403564453125, -0.494140625, -0.241455078125, -2.115234375, 1.2158203125, 0.4345703125, -0.1492919921875, 0.70068359375, 2.078125, -0.2626953125, -1.68359375, -0.15380859375, -0.06005859375, -0.2498779296875, -1.0078125, 0.2493896484375, -0.578125, 1.080078125, -0.6611328125, 1.8359375, -0.317138671875, -0.69677734375, 0.371337890625, -0.77880859375, -0.8212890625, 0.0029697418212890625, -1.197265625, 0.67529296875, -0.164794921875, -1.1533203125, -2.060546875, 0.6533203125, 1.22265625, 1.0859375, 2.208984375, 0.8310546875, 0.57275390625, -0.70849609375, 0.1505126953125, 0.052581787109375, 0.327880859375, 0.357177734375, -2.861328125, 0.1195068359375, -0.0244140625, 0.388427734375, 0.390869140625, 0.3828125, -1.087890625, -0.5830078125, -1.7001953125, -0.1634521484375, -1.2734375, -1.0888671875, 0.45263671875, 1.826171875, -0.47802734375, 1.30078125, 2.421875, 1.5029296875, -1.3369140625, 0.931640625, -0.96435546875, -0.1285400390625, 0.61376953125, 0.56201171875, -0.49462890625, 0.744140625, -0.7353515625, -0.8896484375, 0.07537841796875, 1.5458984375, 0.54296875, -0.6982421875, 1.248046875, 0.2470703125, -0.923828125, 0.92529296875, -0.2301025390625, 1.19140625, 0.03985595703125, 0.352294921875, -0.7548828125, 1.4248046875, 1.732421875, -0.1878662109375, -1.9482421875, -0.85546875, 0.37939453125, 1.126953125, 0.1866455078125, 0.314453125, 0.228759765625, -0.09869384765625, -0.00928497314453125, -0.27685546875, 1.416015625, 0.364990234375, -0.054779052734375, 1.31640625, -1.27734375, 0.72412109375, 0.44677734375, 0.9658203125, -1.0927734375, 0.11334228515625, 0.72607421875, -1.107421875, -2.2890625, -0.8291015625, -0.4736328125, -0.7197265625, 1.677734375, 0.705078125, 1.0322265625, -0.18603515625, -0.52490234375, -1.7197265625, 1.3916015625, -1.5, -1.2373046875, 0.294921875, 0.42578125, 0.64990234375, -0.0684814453125, -0.328369140625, -2.0078125, -0.046783447265625, -0.2113037109375, -0.300048828125, -1.1328125, 0.048675537109375, 1.0244140625, 1.119140625, 1.4072265625, 0.396240234375, -1.2080078125, -0.258056640625, 0.09210205078125, -0.48388671875, 0.21728515625, 0.3818359375, 0.56591796875, 0.29248046875, 0.10467529296875, -1.419921875, -0.251708984375, -0.2861328125, 0.681640625, 0.2152099609375, 0.8173828125, 2.21875, 0.41650390625, 0.1192626953125, -0.025482177734375, 0.209228515625, -0.32177734375, -1.6337890625, -1.412109375, 0.93603515625, 1.0751953125, -1.0771484375, 0.7177734375, -0.337890625, 0.38037109375, -0.74169921875, 0.400390625, -0.04290771484375, 0.80908203125, 0.334716796875, 0.022613525390625, -0.96875, 0.84033203125, -1.1650390625, -0.418701171875, 1.294921875, -0.87890625, -0.04852294921875, 0.00017786026000976562, -2.083984375, -0.10479736328125, 0.861328125, 0.6005859375, -1.2333984375, 0.462646484375, -0.501953125, -1.0361328125, -0.849609375, 1.310546875, -0.07275390625, 0.92822265625, 1.419921875, -0.8759765625, -1.732421875, -0.95263671875, 2.056640625, 1.990234375, 0.254150390625, -0.76904296875, -1.0302734375, -1.65234375, -0.28515625, -0.8251953125, -1.208984375, 0.79931640625, -1.8876953125, 0.68310546875, -1.21875, -0.1231689453125, 0.2301025390625, -0.9072265625, -0.263671875, 1.4052734375, -1.333984375, -2.564453125, 1.921875, -1.7294921875, 0.041107177734375, -0.89306640625, -0.77392578125, -0.026885986328125, -2.015625, 0.95068359375, 0.91015625, -1.546875, 0.470703125, 1.2890625, -0.95703125, -0.373046875, 0.378173828125, -0.198974609375, 0.322021484375, 0.51318359375, 0.8388671875, -0.30712890625, -0.380615234375, 0.051849365234375, -0.1470947265625, -0.82373046875, 1.5517578125, 0.96337890625, -0.81787109375, -1.2177734375, -0.7314453125, -0.38232421875, 0.13720703125, 0.476318359375, 0.501953125, -0.2081298828125, 2.13671875, -0.247802734375, 0.7861328125, 0.093017578125, -0.357666015625, 0.27392578125, -0.5244140625, -1.3720703125, -0.34228515625, -1.63671875, 0.8994140625, -0.58154296875, -1.03125, -0.65576171875, -0.30517578125, 0.01027679443359375, 1.4267578125, 0.658203125, 0.9306640625, 0.59619140625, 0.442138671875, -0.27734375, 0.1455078125, 0.708984375, 0.1884765625, -0.37646484375, 0.359619140625, 2.6484375, -1.8623046875, -1.1005859375, -0.2183837890625, -0.006206512451171875, 0.115966796875, 0.020294189453125, 0.08184814453125, 1.5380859375, -2.603515625, -2.005859375, -0.404052734375, 0.6943359375, -1.3251953125, 0.81640625, -0.485107421875, 0.52490234375, 0.8447265625, 1.01171875, -0.48291015625, -0.018524169921875, -0.50244140625, 0.231201171875, 1.11328125, -1.1337890625, -0.58740234375, -0.83740234375, -1.6923828125, -2.14453125, 1.384765625, 0.35205078125, 0.0015773773193359375, 1.23046875, 0.017303466796875, -2.8671875, -0.501953125, -0.25634765625, -0.66064453125, 1.3974609375, 0.11773681640625, -1.0166015625, 1.279296875, 0.4853515625, -0.75732421875, -0.393798828125, 0.11651611328125, 1.1201171875, -0.408935546875, -0.12115478515625, 0.56689453125, 0.4189453125, -2.103515625, -1.669921875, 0.1651611328125, 0.1243896484375, -0.43359375, -2.748046875, 0.071044921875, 0.95361328125, 0.11834716796875, -1.861328125, 0.5068359375, 0.1552734375, 0.267333984375, 0.6396484375, 0.393798828125, 0.03936767578125, -0.8896484375, -0.1326904296875, -0.6474609375, 0.459716796875, -0.31103515625, -0.00860595703125, -0.421875, 0.1995849609375, -1.0107421875, 0.98095703125, -1.5283203125, -1.353515625, 0.52880859375, -0.6669921875, 0.144775390625, -0.174072265625, 0.036376953125, 0.145751953125, -0.5478515625, 0.23193359375, 0.33251953125, 0.333740234375, 0.54443359375, -0.681640625, -0.29833984375, 1.2646484375, 0.74072265625, -0.32177734375, -2.09375, 0.52392578125, 0.5380859375, -1.001953125, -0.0290374755859375, 1.69921875, -2.07421875, -0.85693359375, -0.900390625, -1.0595703125, 0.143310546875, 1.5966796875, -0.226806640625, -1.939453125, -0.86767578125, -1.0, -0.64208984375, 0.68798828125, 1.630859375, -0.388916015625, 0.0423583984375, -0.751953125, -0.6083984375, 0.00848388671875, -0.4384765625, -0.51806640625, -2.142578125, 0.88037109375, -0.4755859375, 0.95068359375, 1.9658203125, -1.212890625, 1.4423828125, -1.251953125, -0.05999755859375, 0.397216796875, -0.8115234375, 0.04693603515625, 0.1820068359375, -0.75390625, -1.8525390625, -0.751953125, -0.71630859375, 0.120361328125, 1.8251953125, 0.91064453125, -0.444091796875, 0.298095703125, -0.348876953125, 0.498046875, -2.01171875, -0.127197265625, -1.16796875, 0.26953125, -1.6943359375, -0.2359619140625, -1.6259765625, -0.0916748046875, 0.067138671875, 1.5029296875, 0.18017578125, 1.2314453125, -0.0545654296875, 2.0625, 0.75830078125, 1.3193359375, -0.77734375, 1.046875, -0.927734375, -1.4873046875, 0.171630859375, 0.75927734375, 0.64306640625, 1.6962890625, -1.0625, 0.86181640625, -0.250732421875, 0.78515625, 1.1884765625, -0.1273193359375, 1.1767578125, 0.767578125, 0.06951904296875, 0.91845703125, 0.05084228515625, 0.377685546875, -1.2255859375, 0.28369140625, -0.0667724609375, 0.54296875, -1.42578125, -1.626953125, -0.9619140625, -0.042144775390625, -0.213623046875, -1.1572265625, 1.708984375, 0.1552734375, -0.328857421875, 1.248046875, -0.07440185546875, -1.7041015625, 2.35546875, -1.3017578125, -0.234130859375, -0.219970703125, 0.85400390625, 0.488037109375, 0.5244140625, -0.7578125, 0.65185546875, -0.6552734375, 0.5029296875, -1.0712890625, 0.455322265625, -1.8837890625, -1.0126953125, 0.27685546875, 0.8759765625, 0.31494140625, 0.407470703125, -0.1002197265625, 0.8662109375, -0.52197265625, -1.1259765625, -1.015625, 0.0250091552734375, -0.1314697265625, -0.31884765625, -0.114501953125, -0.8056640625, -1.4697265625, 0.1241455078125, 0.5048828125, -1.75, -0.035552978515625, 0.11822509765625, 0.50927734375, -0.931640625, -0.8544921875, -0.6572265625, -0.56103515625, 0.76416015625, 0.432861328125, 0.60546875, -0.58056640625, -1.6826171875, 2.015625, -1.3095703125, 0.46533203125, 0.265380859375, -0.1995849609375, -0.73291015625, 1.748046875, 0.0195465087890625, 1.1982421875, -0.302978515625, -0.65625, 1.130859375, -1.197265625, 0.5546875, 1.427734375, 0.04351806640625, 0.055816650390625, 1.2763671875, 1.2333984375, 1.0556640625, 0.36279296875, 0.337158203125, 0.21337890625, 0.50927734375, -0.681640625, 0.25146484375, 1.3193359375, 2.26171875, 0.1446533203125, -0.43212890625, -0.53857421875, 1.123046875, -1.341796875, -0.7578125, 0.263671875, -0.412841796875, -1.1337890625, -1.001953125, 0.22216796875, 0.482177734375, -0.9609375, 0.640625, -0.62548828125, -1.0419921875, -0.81884765625, -0.04425048828125, -0.408447265625, -0.399658203125, -1.2431640625, 0.078369140625, -0.93798828125, -0.08697509765625, -1.0146484375, -0.180419921875, 0.1199951171875, 0.9443359375, 1.755859375, -0.94921875, 0.5146484375, -0.76171875, -0.88916015625, -0.92041015625, 1.4716796875, 1.13671875, 0.849609375, 0.046173095703125, -1.0048828125, 1.615234375, -0.595703125, 0.324462890625, -0.489501953125, -1.09375, -1.23046875, 1.330078125, -0.1370849609375, 1.7109375, -0.6513671875, -0.11767578125, -0.6376953125, 1.2392578125, 0.62841796875, -0.3994140625, -0.1304931640625, 0.73974609375, -0.5810546875, -0.53076171875, 0.43212890625, -0.90234375, 1.033203125, -1.0419921875, -0.210205078125, -0.8486328125, 2.095703125, 1.5185546875, -1.2138671875, 0.0054168701171875, -0.0185089111328125, -0.050994873046875, -1.1083984375, -0.52392578125, -0.1302490234375, -1.28515625, -0.39404296875, 1.1611328125, -0.41015625, 1.1337890625, -0.72314453125, 0.4853515625, 0.6396484375, -0.6220703125, -0.0833740234375, 1.3037109375, 0.4306640625, -0.67822265625, 0.29833984375, 0.185791015625, 0.79833984375, -0.11865234375, -1.08203125, 0.38916015625, 1.5068359375, 0.11114501953125, 0.06500244140625, -0.1077880859375, -0.2322998046875, -0.703125, -0.36279296875, -0.6923828125, -0.400634765625, -1.4990234375, 0.337158203125, 0.353271484375, -1.09375, -0.1875, -1.505859375, 1.107421875, -0.01178741455078125, 1.6826171875, 0.362548828125, -2.55078125, -0.309814453125, 0.463623046875, -0.97314453125, -0.037628173828125, -1.34375, -0.94677734375, 0.276123046875, 2.7109375, -0.80322265625, -1.0478515625, -0.60546875, -0.341552734375, 0.607421875, -0.264404296875, -0.62158203125, -0.05499267578125, -1.1982421875, -1.2734375, 0.51171875, 0.410888671875, 1.1591796875, -0.7578125, -2.330078125, 0.86083984375, 0.48193359375, 0.0211181640625, 1.50390625, -0.336669921875, 0.08856201171875, 1.6357421875, -0.88427734375, -2.580078125, -2.09765625, 1.5205078125, -0.0187835693359375, 1.0537109375, 0.31298828125, 0.63232421875, 2.203125, -2.63671875, 0.1373291015625, 0.93212890625, -0.8994140625, -0.070068359375, 0.267578125, -0.30810546875, 0.056640625, -0.294921875, 0.75634765625, 1.08984375, -2.078125, 1.29296875, -2.693359375, -1.1376953125, 0.8310546875, -0.0230865478515625, -1.0986328125, -0.7158203125, -0.88134765625, -0.7216796875, 0.8291015625, -0.041107177734375, -1.259765625, 0.5888671875, 0.22216796875, 0.279541015625, 0.34521484375, 1.5966796875, 0.85400390625, -2.177734375, 1.14453125, -0.285888671875, 1.3818359375, -0.251220703125, -0.212158203125, -0.51123046875, -0.26025390625, -0.05926513671875, 1.9052734375, -1.041015625, 0.87255859375, -0.274658203125, 0.521484375, 0.159423828125, -0.12237548828125, 1.25, -2.4609375, 0.83740234375, -0.005428314208984375, 1.6953125, -0.26123046875, 0.9033203125, -0.6650390625, 0.326171875, -1.400390625, -0.458984375, -1.0185546875, 0.134521484375, 2.12109375, 0.64404296875, 1.0400390625, -0.053192138671875, 0.39990234375, 0.2100830078125, -1.48046875, -0.70166015625, 0.6953125, 0.441162109375, -1.9306640625, 2.685546875, 2.150390625, 0.267333984375, -0.1312255859375, -0.5634765625, 1.125, -0.7822265625, 1.3544921875, -0.375244140625, -0.05584716796875, -0.1644287109375, -1.0185546875, -0.92333984375, -0.11724853515625, -0.480712890625, -2.27734375, -0.4267578125, 1.537109375, -1.4228515625, 0.1708984375, 1.8984375, 0.032012939453125, -0.28955078125, -1.251953125, -0.12286376953125, -0.38232421875, 0.67236328125, -0.201904296875, -0.63037109375, -0.9736328125, -2.130859375, -0.537109375, 0.7587890625, -1.5966796875, -0.64013671875, -1.1103515625, 1.595703125, -0.0107879638671875, 0.60595703125, 0.7236328125, 0.96630859375, -1.4599609375, 0.0770263671875, 1.2138671875, -1.05078125, -0.372802734375, 1.197265625, 0.2222900390625, 0.2900390625, 0.75732421875, 1.2177734375, -0.63037109375, 1.3740234375, 0.6142578125, 0.16162109375, -0.83251953125, 0.389892578125, -0.164794921875, -0.8095703125, 1.3212890625, 0.67041015625, -1.71875, -1.0693359375, -2.095703125, -1.2080078125, -0.1885986328125, -0.548828125, -0.51318359375, 0.022369384765625, -1.5361328125, -0.2705078125, -0.0462646484375, -1.259765625, -0.89599609375, 1.126953125, -0.9296875, 0.421142578125, -0.12060546875, -0.6787109375, -0.693359375, 0.99072265625, -0.69677734375, 0.55908203125, -0.84814453125, -1.580078125, -0.748046875, 0.4150390625, 1.060546875, -1.3330078125, -1.53515625, -0.92138671875, -0.079833984375, 0.98779296875, -1.93359375, -1.2900390625, 0.6435546875, -0.217041015625, 0.316162109375, 0.74951171875, -0.60009765625, 0.5302734375, -2.041015625, -0.4404296875, 0.5390625, -0.51806640625, 0.57177734375, 0.440673828125, -0.82373046875, 1.775390625, -0.52734375, -0.349365234375, 0.05084228515625, -0.2467041015625, -0.78759765625, -0.2132568359375, -1.24609375, -1.2646484375, -1.408203125, 0.740234375, 1.5986328125, -0.75244140625, -0.276123046875, 2.619140625, -0.72119140625, 0.298583984375, -1.80859375, -1.052734375, -0.591796875, -0.6064453125, 0.4921875, 0.8359375, -0.0999755859375, 0.3388671875, 1.0078125, 0.3056640625, 0.708984375, 0.414306640625, 1.361328125, 1.10546875, 0.438232421875, 1.302734375, 1.54296875, -0.153564453125, -2.8359375, 0.69189453125, -0.263916015625, -0.2509765625, 0.9736328125, -0.86962890625, -0.6611328125, -0.58251953125, -1.392578125, -1.7705078125, 1.0234375, -0.14404296875, 0.81640625, -0.02532958984375, -0.1866455078125, -0.272705078125, 0.1578369140625, -0.96923828125, 0.65234375, 1.65625, -0.246826171875, -0.80078125, -0.04180908203125, 0.281005859375, -0.611328125, 0.46435546875, -2.03515625, -0.947265625, 0.58056640625, -1.5771484375, -1.8115234375, 0.1324462890625, -0.2410888671875, 1.525390625, 1.6171875, -1.400390625, 0.0301666259765625, 0.370361328125, -0.5859375, -0.62646484375, -0.71875, -1.0712890625, 0.49169921875, 0.178955078125, -1.0009765625, -0.57275390625, -0.58740234375, -0.56689453125, 1.0224609375, 0.216552734375, 0.8486328125, -0.1976318359375, 0.9140625, -0.0233917236328125, 1.4443359375, 1.0361328125, 0.982421875, 0.1885986328125, 0.302001953125, 1.0810546875, 0.6005859375, -0.494140625, -0.140380859375, 2.140625, -0.68115234375, 0.64794921875, -1.3173828125, -1.154296875, -2.0078125, -1.01171875, -1.7666015625, 2.412109375, 0.09100341796875, -0.9619140625, 1.5947265625, -0.84912109375, -0.76025390625, 0.79736328125, 1.388671875, 0.014190673828125, 0.75146484375, -0.8134765625, 0.0491943359375, -1.8671875, 0.72265625, 0.86181640625, 2.244140625, -0.73388671875, 1.6865234375, -0.4951171875, 2.50390625, 0.0833740234375, 0.80078125, 0.68212890625, -1.4423828125, 0.5400390625, 0.236572265625, 0.06951904296875, -0.394287109375, -0.242431640625, 0.2218017578125, 1.306640625, -0.98486328125, -0.96728515625, -2.845703125, 0.24951171875, -1.048828125, 2.619140625, -0.55712890625, 0.281005859375, 0.366943359375, -0.6201171875, 0.13525390625, 0.1124267578125, -0.56982421875, 0.66943359375, -0.7099609375, 0.441650390625, -0.56494140625, 0.10498046875, -0.1837158203125, 2.4609375, -2.033203125, 0.425537109375, 0.58251953125, 0.368896484375, 0.24462890625, 0.2113037109375, 0.62158203125, 1.587890625, -0.230712890625, 1.78515625, 0.281005859375, 1.1689453125, 0.142578125, 1.154296875, 1.5224609375, -0.2158203125, -0.169189453125, -0.0426025390625, -0.299072265625, -0.5048828125, 0.61181640625, -0.83740234375, 1.6025390625, 0.245361328125, 0.315673828125, 0.416259765625, 0.5634765625, -1.5986328125, -0.98095703125, -0.318115234375, -0.88037109375, -2.1875, 0.006855010986328125, 0.81298828125, 1.1943359375, 1.3115234375, -1.5107421875, -0.5478515625, 0.22265625, -0.11712646484375, 1.189453125, -0.0977783203125, -0.324462890625, 0.72119140625, -1.396484375, -0.0097503662109375, -0.218505859375, -0.77880859375, -0.31005859375, -0.13671875, 0.273193359375, 1.728515625, 0.469482421875, -0.6953125, -0.371826171875, -0.6318359375, -0.1810302734375, -0.8603515625, -0.64111328125, -1.810546875, 0.1669921875, -0.48681640625, 0.1829833984375, -0.332275390625, -1.568359375, -0.1224365234375, -0.6533203125, 1.7431640625, 0.2430419921875, 1.5712890625, 0.57666015625, -0.294189453125, -0.256103515625, 0.80078125, -0.284423828125, 0.26708984375, 0.435546875, 0.390380859375, 0.59326171875, -1.015625, -0.198486328125, 3.28125, 0.93359375, 0.022796630859375, 0.1231689453125, 0.1527099609375, -2.966796875, 0.98876953125, -0.72314453125, -0.89453125, -0.48486328125, -0.2303466796875, -0.340576171875, 1.359375, -0.106689453125, -0.95703125, -0.166259765625, 0.67333984375, 0.3974609375, 0.2012939453125, 0.59375, -0.83349609375, -0.50244140625, -1.568359375, -1.623046875, 1.87890625, -0.994140625, 0.8134765625, -0.9658203125, -0.1461181640625, -0.434814453125, 0.2406005859375, -0.56591796875, 1.642578125, 0.73681640625, -0.0733642578125, 0.70361328125, -1.333984375, 1.1328125, 0.361083984375, -0.2431640625, -0.268798828125, 0.0005269050598144531, 0.59814453125, -0.89697265625, 1.154296875, 0.52197265625, 1.3828125, 1.455078125, 0.82470703125, 1.0087890625, 1.2060546875, 1.251953125, -0.03125, -2.40234375, -0.90673828125, 0.074462890625, -0.71044921875, -1.6552734375, 0.1119384765625, -1.6083984375, -1.6376953125, -1.666015625, 0.0849609375, 2.18359375, 0.428466796875, 1.0322265625, -1.2744140625, 0.059478759765625, -0.261962890625, 2.265625, -1.28515625, -0.09893798828125, 0.389404296875, -0.873046875, -0.01104736328125, 1.625, 1.2734375, -1.005859375, -0.0792236328125, 0.076416015625, 0.53271484375, -0.01517486572265625, 1.556640625, 0.88671875, 0.681640625, 1.75390625, -1.8017578125, 0.57470703125, -1.572265625, -1.11328125, 0.83935546875, 1.658203125, -0.2115478515625, 0.67431640625, 0.363037109375, 0.67236328125, -0.138671875, 1.4521484375, -0.09912109375, 0.6953125, 1.166015625, 0.126953125, -0.43115234375, 0.151611328125, -0.78857421875, -1.291015625, 1.521484375, -0.2568359375, 1.5751953125, -0.135498046875, 0.13427734375, 1.615234375, -1.0087890625, -1.9892578125, 0.6435546875, 0.256103515625, 2.138671875, 1.09765625, -0.31640625, -0.320556640625, -0.93115234375, -0.1796875, -0.56103515625, 1.2861328125, -1.380859375, -0.405029296875, 0.377685546875, -1.572265625, -2.033203125, -0.385986328125, -1.7529296875, 0.206298828125, -1.19140625, 0.68310546875, 0.33154296875, 0.35888671875, 1.666015625, 0.5419921875, -1.828125, -0.400634765625, -0.335693359375, 0.1685791015625, 0.93310546875, -0.1903076171875, -0.6484375, -0.6279296875, -0.91455078125, 0.96337890625, -0.720703125, -1.7900390625, -0.0673828125, 0.89404296875, -1.2099609375, -0.2113037109375, -1.1748046875, -1.353515625, 1.755859375, -0.96630859375, 0.1983642578125, 0.5419921875, -0.383056640625, -0.86572265625, -0.486083984375, -0.8701171875, -1.431640625, -0.5546875, 1.0986328125, -0.1390380859375, -1.1298828125, -0.01404571533203125, -0.1871337890625, 0.91259765625, -0.495361328125, -1.677734375, -1.39453125, 0.341064453125, -1.1533203125, -0.45068359375, 1.1025390625, 0.11376953125, 1.0205078125, -1.244140625, -2.369140625, 2.58203125, -0.140869140625, 0.98291015625, -0.2176513671875, -0.7275390625, -1.822265625, -0.1988525390625, -0.1119384765625, 0.6513671875, -0.55859375, 1.1630859375, 1.3505859375, 1.234375, 1.6796875, -0.48583984375, 0.299072265625, 0.66015625, -0.029205322265625, 0.162353515625, 1.6279296875, -0.01049041748046875, 0.0865478515625, 0.287353515625, -2.154296875, 0.9580078125, 0.6630859375, -0.2076416015625, -0.93603515625, 0.5390625, 0.19580078125, 1.13671875, 0.533203125, -1.482421875, 0.73291015625, -0.1484375, 1.7900390625, 0.56005859375, 1.0908203125, -0.11260986328125, -0.5732421875, 0.5751953125, -0.2113037109375, 1.3994140625, -0.03533935546875, 0.324462890625, -2.0546875, -1.384765625, 0.69140625, -0.270263671875, -0.434326171875, 0.7607421875, 1.150390625, -0.206298828125, -0.90087890625, 3.333984375, -0.70703125, 0.289794921875, -0.465576171875, -1.171875, -0.78515625, 1.35546875, -1.9306640625, -1.3974609375, 0.2646484375, 1.62109375, -0.55224609375, 0.1324462890625, 0.236572265625, 0.861328125, 0.64306640625, 0.810546875, -0.9052734375, -0.07147216796875, -0.599609375, -1.740234375, 1.0146484375, 1.341796875, -0.1177978515625, 1.34375, 0.399658203125, 0.72412109375, -0.364501953125, 0.88232421875, 0.050689697265625, -0.428466796875, 0.388427734375, -1.1318359375, -0.1790771484375, -0.56982421875, 1.83984375, -0.826171875, 2.392578125, 1.396484375, 0.7275390625, -1.8994140625, 1.0341796875, 0.525390625, -1.9296875, 0.11944580078125, 1.6220703125, -0.3564453125, -1.3134765625, 1.8798828125, 1.8115234375, 0.9697265625, -0.2783203125, -0.1446533203125, 0.391845703125, -0.7568359375, 0.01800537109375, -0.54248046875, 0.359130859375, 0.533203125, -0.43603515625, 1.0791015625, -1.1923828125, 0.75634765625, 0.97021484375, 1.3828125, -0.04656982421875, -0.367431640625, 1.9716796875, -0.83642578125, -1.2666015625, 0.763671875, 0.060150146484375, 1.4306640625, -0.034820556640625, -0.443359375, 0.0780029296875, -0.77099609375, 0.396240234375, -0.50927734375, -1.0029296875, -0.1988525390625, 0.697265625, 1.3046875, 0.19384765625, 0.6787109375, 0.64453125, -0.2197265625, 0.1639404296875, -0.276611328125, 1.03515625, -0.010284423828125, -0.1549072265625, 2.4140625, 0.4443359375, -1.4951171875, -0.01885986328125, -0.73779296875, 0.2919921875, 1.3505859375, 1.0166015625, -1.1044921875, 0.64794921875, 0.408935546875, -0.431884765625, 0.8955078125, 0.238525390625, -0.360107421875, 0.34375, -1.1484375, -1.4306640625, -0.033050537109375, -0.29443359375, -0.57568359375, -0.34228515625, -0.9677734375, -0.183349609375, 1.2412109375, 0.4130859375, -2.16796875, -0.55859375, 0.08050537109375, -2.162109375, -0.2493896484375, 0.8544921875, -0.78076171875, 0.716796875, 0.97998046875, 0.467041015625, -0.017364501953125, 0.3642578125, 0.182373046875, -0.475341796875, -0.00970458984375, -0.165771484375, -0.56982421875, 0.65234375, -0.71630859375, -1.0986328125, 0.284423828125, -1.0595703125, 0.89013671875, 0.74462890625, 1.20703125, 0.352783203125, 0.513671875, -1.2978515625, 0.2900390625, -1.712890625, 0.1182861328125, 1.095703125, 1.1875, 0.9814453125, -0.473388671875, -0.8916015625, 1.3330078125, 0.4169921875, 0.2841796875, 0.09619140625, -0.77099609375, 0.477294921875, -0.82373046875, -0.3740234375, 0.5244140625, -1.62890625, -0.353515625, -2.123046875, 0.034881591796875, -0.5400390625, 0.65185546875, -0.798828125, 0.265380859375, 0.1939697265625, 1.189453125, -0.10211181640625, 1.6220703125, -0.2218017578125, 0.26318359375, 1.04296875, 0.86474609375, 1.1875, 2.294921875, 1.4873046875, 0.72216796875, -0.206298828125, -0.1402587890625, 0.05126953125, -0.48876953125, 0.8251953125, -0.48681640625, 0.77294921875, -1.046875, 0.61865234375, -0.6962890625, -1.3515625, -1.1220703125, -1.6259765625, 0.58642578125, 1.0185546875, 0.916015625, 0.75439453125, 1.2529296875, -0.517578125, 0.10833740234375, -0.52880859375, -0.244384765625, 1.01953125, -0.875, -0.055999755859375, 1.2919921875, 0.6455078125, 1.2890625, -0.8388671875, -0.09515380859375, -1.3798828125, 0.833984375, 0.91259765625, -1.236328125, -1.25390625, 1.60546875, 0.14697265625, 0.70166015625, -0.3642578125, 0.58203125, 0.2293701171875, 1.3984375, 2.259765625, 0.7568359375, 1.9140625, -0.57080078125, -0.70068359375, -1.345703125, -0.2076416015625, 0.28271484375, -2.443359375, 0.37939453125, -0.00775146484375, 0.6201171875, -0.7021484375, -0.07342529296875, 0.62451171875, -0.294189453125, -0.2396240234375, -1.5986328125, -0.642578125, 0.313232421875, 0.10028076171875, -0.04461669921875, 0.66845703125, -0.3203125, 0.76123046875, 2.1875, 0.1212158203125, -1.8076171875, 2.072265625, -0.10284423828125, 0.25439453125, -0.11767578125, 0.98681640625, -0.6376953125, -0.2353515625, -0.331787109375, -0.43505859375, 0.10797119140625, 0.552734375, 1.623046875, 1.1064453125, 0.417236328125, 0.005138397216796875, -0.8193359375, -1.0078125, 2.794921875, -1.6005859375, -0.0877685546875, -1.623046875, 0.89599609375, 1.150390625, -1.140625, -0.493896484375, -0.7568359375, 0.0799560546875, 0.4267578125, -0.90625, -0.174072265625, 1.791015625, 0.284423828125, 1.091796875, 0.79638671875, -0.72607421875, -1.5009765625, -0.7841796875, 0.260498046875, 0.861328125, 0.34033203125, -1.4755859375, 1.1064453125, 0.69921875, -0.1390380859375, 0.1964111328125, 0.8955078125, 2.216796875, 0.1263427734375, 0.10028076171875, 0.0948486328125, 0.10528564453125, 1.32421875, -2.314453125, 0.63525390625, 1.6640625, -1.5537109375, -0.258544921875, 0.65771484375, 0.395263671875, -1.2607421875, 0.252197265625, -0.2044677734375, -0.10577392578125, 1.6474609375, 0.86328125, -0.53125, -2.484375, 1.8095703125, -0.326171875, -0.69287109375, -0.986328125, -0.60498046875, -2.490234375, -0.91455078125, 0.373779296875, 0.7177734375, -1.720703125, 0.47216796875, -0.71337890625, 1.1669921875, -0.7919921875, 0.35546875, -0.60302734375, -0.763671875, 0.59423828125, 1.052734375, -0.0168304443359375, -0.84619140625, 0.7587890625, 0.251953125, 1.810546875, 0.248046875, -1.1279296875, 2.212890625, -0.626953125, 0.71923828125, -0.642578125, -1.0283203125, 0.1822509765625, 1.27734375, -0.94580078125, -0.81201171875, 1.138671875, 1.138671875, 1.4892578125, 0.1539306640625, -1.0927734375, 0.7568359375, -0.5830078125, -0.29443359375, 0.91455078125, -0.50048828125, 1.3408203125, 1.341796875, -0.61376953125, 0.0226287841796875, 0.33447265625, -0.74267578125, 0.324951171875, -1.185546875, 1.1318359375, 1.3642578125, 0.416015625, -1.248046875, -1.2041015625, 1.509765625, -0.84619140625, -1.9169921875, 1.001953125, -0.64892578125, -0.22802734375, 0.385498046875, 0.049530029296875, -0.38427734375, -0.037017822265625, 0.52587890625, 1.416015625, -0.431884765625, 0.377685546875, -0.10394287109375, -0.158447265625, 0.93994140625, -0.037384033203125, -1.28515625, -0.8818359375, 1.279296875, -1.0869140625, -1.1494140625, -0.64892578125, -2.984375, 0.8017578125, -0.274658203125, 0.1959228515625, -0.9853515625, 1.8857421875, 0.42431640625, -0.970703125, -0.2705078125, -0.1854248046875, -1.1142578125, -0.284912109375, 0.53759765625, 0.283203125, 0.990234375, -0.17919921875, -1.2529296875, -1.0029296875, -0.455810546875, 0.274658203125, -0.47509765625, 1.1044921875, -0.227294921875, -0.78564453125, 0.5048828125, -0.353515625, 0.87255859375, -1.4423828125, -0.82763671875, -0.204833984375, 1.8076171875, 0.2003173828125, 0.9375, 0.277587890625, 0.1695556640625, 0.1689453125, -0.99609375, 0.322998046875, -1.60546875, 1.4404296875, -0.3154296875, 0.1251220703125, -0.007038116455078125, 0.59228515625, -1.046875, 1.68359375, -0.1693115234375, -1.123046875, 0.5322265625, -0.65380859375, -0.73486328125, -1.84765625, -0.48046875, 0.9111328125, -0.91650390625, 0.0732421875, 0.81201171875, 0.94482421875, 0.037017822265625, -0.55712890625, -0.6875, -0.403076171875, 0.9052734375, 1.1875, 0.471923828125, 0.07208251953125, 0.04315185546875, 1.2724609375, -0.7216796875, -0.392333984375, 0.1514892578125, -1.0830078125, 0.01349639892578125, 0.9228515625, 2.1328125, 0.72314453125, 0.69287109375, -1.875, -0.0311126708984375, 0.1365966796875, -0.357666015625, 0.2015380859375, -0.73974609375, -0.301513671875, -0.1668701171875, -0.346435546875, -0.44921875, 0.6181640625, 0.31689453125, 0.269287109375, -0.86962890625, -0.78955078125, 1.791015625, -0.57080078125, 0.654296875, 0.90869140625, 0.900390625, -0.59912109375, -1.38671875, 0.75244140625, -1.310546875, 1.4130859375, -1.0986328125, 1.56640625, 0.5751953125, 1.2646484375, 0.7314453125, 1.7783203125, -3.427734375, -2.73046875, 0.9736328125, 0.350341796875, 0.1126708984375, -1.552734375, -1.484375, 1.185546875, 1.3427734375, 0.348876953125, -1.66796875, -0.467041015625, -1.3173828125, 0.70263671875, -0.94775390625, -1.0615234375, -0.84326171875, 0.1466064453125, -1.0927734375, -0.060791015625, 0.1072998046875, 0.57373046875, -0.386474609375, -0.287353515625, -1.81640625, -0.358154296875, -1.1806640625, -0.036895751953125, -1.27734375, -0.342529296875, 0.0955810546875, -1.8603515625, 1.2177734375, -0.44677734375, -0.87939453125, -1.0205078125, 0.59228515625, -0.9462890625, 0.1175537109375, -0.96142578125, -0.042144775390625, 1.0224609375, -0.53125, -0.142333984375, -0.716796875, -0.82421875, -0.1224365234375, -0.5302734375, -0.2587890625, 1.052734375, 0.07098388671875, 0.290283203125, 2.373046875, -3.82421875, -1.0263671875, 2.234375, -1.79296875, -0.62353515625, -0.70068359375, -0.67041015625, 0.0880126953125, 0.312744140625, 0.05133056640625, -0.86181640625, 0.08740234375, 0.85693359375, -0.63330078125, -0.09564208984375, -1.515625, -0.12493896484375, 0.4541015625, 1.5146484375, -0.2227783203125, 0.57470703125, -0.473876953125, 0.2744140625, 0.2012939453125, -0.317138671875, 1.3701171875, -0.1009521484375, 2.337890625, -0.1966552734375, -0.90234375, -0.1514892578125, 0.264404296875, 1.7705078125, 0.5927734375, -0.4189453125, -0.87353515625, -0.047821044921875, -1.9970703125, 0.763671875, -1.65234375, 0.9482421875, 0.7333984375, -0.4765625, 0.2322998046875, 1.173828125, -0.88525390625, -0.314453125, 0.63720703125, 0.473388671875, -0.1341552734375, -0.444580078125, 0.12255859375, 0.402587890625, -0.02777099609375, -1.16015625, 0.2467041015625, 0.59912109375, -1.1796875, 1.2421875, -0.5859375, -0.69970703125, 0.85400390625, 0.5126953125, 1.0966796875, 0.82861328125, -0.6484375, 0.83935546875, 1.75, 0.258544921875, 0.81884765625, -1.138671875, 0.76708984375, -0.1748046875, 0.04437255859375, -1.1982421875, 0.269287109375, -0.576171875, -0.481689453125, 1.0703125, 0.5791015625, 0.340087890625, 0.329833984375, 1.677734375, -1.0302734375, 0.2191162109375, 0.5263671875, 0.4501953125, -0.73876953125, 0.2388916015625, -1.248046875, 0.41357421875, 0.97216796875, -0.39306640625, 0.236328125, 0.41796875, -1.2646484375, -3.076171875, 0.26025390625, -0.953125, 0.422607421875, -1.443359375, -0.294921875, 1.4169921875, -0.61865234375, -0.6875, -0.15087890625, 0.599609375, 1.6640625, 0.60791015625, -1.3193359375, 0.26513671875, 2.025390625, -1.216796875, -1.7958984375, -0.049163818359375, 0.335693359375, 0.5830078125, 0.11102294921875, 0.155517578125, 0.52783203125, -0.10321044921875, 0.748046875, 0.35009765625, 0.31396484375, 0.420654296875, 0.114013671875, 1.0322265625, 2.66015625, 1.87890625, -0.6875, 1.248046875, -0.19384765625, -0.36181640625, -0.0215911865234375, -0.35791015625, -0.76806640625, 0.12493896484375, -0.051055908203125, 1.2412109375, -1.1298828125, 1.6357421875, -0.68408203125, -0.10491943359375, -2.224609375, 0.2257080078125, -0.177001953125, 0.576171875, 1.4111328125, 1.2685546875, -0.0498046875, -0.06475830078125, 0.1600341796875, 0.46435546875, 0.83056640625, 1.6162109375, 1.4326171875, -1.1337890625, 2.65625, -0.76611328125, -1.48828125, -0.45458984375, 0.035614013671875, 0.744140625, -0.54345703125, -1.1015625, -0.6728515625, -0.0012788772583007812, -0.6298828125, 0.81005859375, -0.7939453125, -1.326171875, -1.6630859375, 0.1734619140625, 0.58203125, -2.287109375, 0.9091796875, -1.0087890625, -0.241455078125, -2.904296875, -0.8759765625, -0.82763671875, -0.6728515625, -1.01953125, -2.287109375, -0.445556640625, -0.212890625, 0.79736328125, -0.55517578125, -1.1201171875, -0.455322265625, -0.486572265625, 1.62109375, 0.32568359375, 0.5732421875, 0.132080078125, -1.3857421875, -1.2978515625, 0.54833984375, -0.026885986328125, -0.01325225830078125, -0.1959228515625, 0.447998046875, -0.046600341796875, 1.0078125, -0.50048828125, 2.1171875, 0.8203125, 1.4189453125, 0.79345703125, 0.0291900634765625, -0.78076171875, 2.5, 2.208984375, 0.072509765625, -0.92724609375, 1.1044921875, 0.304443359375, 1.228515625, -0.50341796875, 0.31494140625, -0.90673828125, 0.090087890625, 1.55078125, 0.0274658203125, -1.0146484375, -0.405029296875, 0.10272216796875, 1.2978515625, -0.98291015625, 0.06451416015625, 0.65087890625, -1.26953125, 0.908203125, 1.6005859375, 0.65869140625, 1.2490234375, 1.0693359375, -0.30908203125, -2.212890625, 1.142578125, 0.76318359375, -0.012542724609375, -1.2275390625, -0.08294677734375, 0.99267578125, 0.0012712478637695312, -1.033203125, -0.421875, 0.634765625, 1.9619140625, -1.43359375, -0.89453125, 1.9375, -0.83740234375, 2.798828125, -0.77880859375, 0.329345703125, -0.04595947265625, 1.2998046875, 2.02734375, 0.1463623046875, -1.177734375, -0.5732421875, 1.1181640625, 0.71533203125, 0.9443359375, 0.10101318359375, 1.7841796875, -0.2176513671875, -0.46728515625, -0.494873046875, -0.11297607421875, 2.02734375, 0.34326171875, -0.24853515625, 1.228515625, -1.1923828125, 0.2156982421875, 1.6591796875, 0.06597900390625, 1.8125, -0.615234375, 0.40283203125, -2.400390625, 0.020538330078125, 2.83203125, -0.40185546875, -0.05133056640625, -0.55322265625, 0.67724609375, -0.875, 0.248779296875, -1.212890625, 0.406982421875, -0.486328125, 0.4736328125, 1.3046875, -1.1513671875, -0.391357421875, 0.4775390625, 0.52734375, 0.578125, -1.5849609375, 0.83837890625, 0.8818359375, 0.7607421875, 0.0977783203125, -0.56787109375, -0.60498046875, 1.791015625, -1.28515625, -0.155517578125, -0.0906982421875, -1.236328125, 0.149169921875, -0.006103515625, -0.7744140625, -0.49365234375, 0.96240234375, -0.8173828125, 0.364013671875, -0.6376953125, 0.290283203125, -0.65234375, 1.1953125, -1.3310546875, -0.472900390625, 0.85205078125, -0.56494140625, -0.330078125, -0.95849609375, 0.32470703125, 0.06121826171875, -1.3681640625, 1.0205078125, 4.2421875, 1.5966796875, -1.6376953125, 1.1796875, 0.7978515625, -0.376708984375, -0.097900390625, 1.1025390625, 1.4609375, -0.95361328125, -0.151123046875, 0.0994873046875, 1.685546875, -0.2318115234375, -0.73681640625, -1.4833984375, -0.7265625, 1.24609375, 0.35205078125, -0.09442138671875, -1.9150390625, 1.8603515625, -0.2054443359375, 0.446044921875, -0.35205078125, -0.90576171875, 0.0021114349365234375, 0.69140625, 0.07379150390625, -0.004405975341796875, -0.6806640625, 0.6376953125, 0.02899169921875, -1.3046875, 1.923828125, 0.8935546875, -1.193359375, -1.16796875, -0.12371826171875, 1.4541015625, -2.490234375, 0.51953125, -1.046875, 0.53515625, -1.115234375, -0.65087890625, 0.7607421875, 0.196533203125, -0.8291015625, -0.238525390625, -0.433837890625, 0.1663818359375, 0.57177734375, 0.50244140625, 0.10614013671875, 0.65185546875, -0.190185546875, 0.345947265625, -0.2744140625, -1.3193359375, -0.38427734375, 0.1710205078125, -0.79833984375, 0.3271484375, -0.042236328125, 0.32421875, 1.2802734375, -1.7890625, 0.845703125, -1.3916015625, 1.7861328125, -0.156005859375, -0.67041015625, -1.0263671875, 0.431640625, -0.345458984375, -2.517578125, 1.328125, -0.869140625, 0.250732421875, -0.92626953125, 0.2020263671875, -0.3671875, 0.8857421875, 2.501953125, 0.10870361328125, 0.1854248046875, -0.73681640625, -0.07147216796875, 1.1533203125, 0.7900390625, 1.0771484375, 2.564453125, -0.156005859375, -0.10247802734375, -1.044921875, -1.23828125, -0.11700439453125, -1.373046875, 0.07562255859375, -1.5263671875, -0.0262603759765625, 0.11163330078125, 0.7119140625, 0.87939453125, 0.7646484375, -1.16015625, 0.958984375, -0.86572265625, -0.87939453125, -1.1171875, -0.8525390625, 0.9619140625, 1.2275390625, -1.189453125, -0.58203125, 0.425048828125, 0.8701171875, 0.72119140625, 0.491943359375, -1.2900390625, -0.52490234375, -1.7861328125, -0.0401611328125, 1.568359375, 0.455078125, -0.68505859375, 1.078125, 1.107421875, 0.5908203125, -2.185546875, -0.54638671875, 0.1995849609375, 0.640625, 1.005859375, 1.44140625, -0.9326171875, -0.51416015625, -0.2342529296875, 1.1259765625, 1.0078125, 0.3515625, 0.326904296875, 0.355224609375, -0.34326171875, -0.78955078125, 0.91552734375, 1.48046875, -0.47705078125, 0.12481689453125, 1.0634765625, 1.66015625, -0.9599609375, -0.19775390625, -0.029632568359375, 1.6767578125, 0.1708984375, -0.2039794921875, 1.67578125, 1.181640625, 1.3544921875, 1.6650390625, -1.0595703125, 1.6083984375, 0.9716796875, 1.96875, -0.3681640625, 0.453369140625, -0.83154296875, -0.93505859375, -0.275146484375, -0.480712890625, -0.07342529296875, 0.37890625, -0.3896484375, -1.193359375, -0.97119140625, -0.3994140625, -0.71728515625, -0.1507568359375, 0.3818359375, 0.69384765625, 1.53125, 2.296875, 0.6484375, 1.3486328125, -0.25390625, 0.36669921875, 1.3330078125, -0.364501953125, -0.83203125, -1.0498046875, -0.79443359375, 0.64013671875, -0.60498046875, 0.2313232421875, -1.248046875, 0.59716796875, -0.822265625, 1.28515625, -0.9169921875, -0.1982421875, 0.14404296875, -0.646484375, 1.50390625, 0.77734375, 0.84765625, 0.017822265625, 1.4697265625, 1.1689453125, 0.68701171875, 1.0234375, -0.1544189453125, -0.323486328125, -0.0626220703125, 1.75390625, 1.0625, -0.38818359375, -1.0986328125, -0.476806640625, -0.211181640625, -1.1953125, 2.25390625, 2.123046875, 0.79443359375, -0.5380859375, -0.08282470703125, 0.52587890625, -0.1663818359375, -0.460205078125, -0.3232421875, 0.4326171875, 1.35546875, 1.033203125, -0.46826171875, 0.38525390625, -1.654296875, 0.55224609375, 0.00592041015625, -1.13671875, 0.12646484375, -0.53125, 1.240234375, 1.880859375, 1.0771484375, 0.5146484375, 0.12457275390625, -0.86474609375, -2.556640625, 0.296875, 1.80078125, -0.86181640625, -1.8876953125, 1.25, 1.39453125, -0.6875, -0.3642578125, -0.06365966796875, 0.98681640625, -0.39404296875, -0.36767578125, -0.0882568359375, -0.72802734375, 0.83203125, -0.0556640625, 1.755859375, -0.7861328125, -0.3193359375, 0.373779296875, -0.64697265625, 2.14453125, -0.1932373046875, 0.63818359375, 0.3232421875, 0.07452392578125, 0.638671875, 0.36865234375, 1.1767578125, -0.8369140625, 1.76171875, -1.1875, 2.029296875, 0.90576171875, 0.68603515625, 1.482421875, -1.830078125, 0.87548828125, 1.0947265625, 0.1607666015625, -0.04571533203125, -0.061004638671875, 3.044921875, -0.79833984375, -0.1256103515625, 1.279296875, -0.2919921875, -1.716796875, -0.47998046875, -0.395751953125, 0.578125, 0.92333984375, -1.974609375, -0.43212890625, 0.74951171875, 0.11639404296875, 0.59521484375, 0.26416015625, -0.26611328125, -1.2021484375, 0.0008897781372070312, 1.224609375, 1.1455078125, 2.123046875, -0.54736328125, -2.63671875, 0.9990234375, 0.1829833984375, -0.68701171875, 0.93505859375, 0.258056640625, -0.84375, -0.81103515625, 0.78564453125, -1.353515625, -1.8291015625, -0.362060546875, -0.52294921875, 1.0966796875, 0.88916015625, -0.9287109375, -1.474609375, -0.07666015625, -0.59912109375, 1.8642578125, -1.0615234375, -0.85888671875, 0.8583984375, -0.0222320556640625, 0.166259765625, -0.79638671875, 1.0791015625, 0.8642578125, 1.115234375, -0.06268310546875, 1.3466796875, 0.06878662109375, -0.1368408203125, -0.75341796875, 0.78857421875, 1.1259765625, 1.013671875, 0.576171875, 1.259765625, 0.354736328125, 0.31591796875, -0.0286102294921875, 1.9404296875, -2.58984375, -0.177490234375, 0.0989990234375, -0.32861328125, 0.273193359375, -1.0927734375, 1.6005859375, 1.24609375, 0.94482421875, -0.70703125, -0.151123046875, -0.01554107666015625, -0.78955078125, -0.79150390625, 1.2529296875, 1.33203125, 2.11328125, 0.0305023193359375, -0.273193359375, -0.271240234375, -1.1357421875, -0.397705078125, 0.76904296875, 0.64697265625, -0.053680419921875, -0.055908203125, -0.412841796875, -0.63525390625, 0.487060546875, 1.48828125, 0.96435546875, -0.1102294921875, -0.72607421875, -0.369140625, 1.0498046875, 1.005859375, -1.5751953125, -2.01171875, 0.61962890625, 1.6982421875, 0.95458984375, -1.5732421875, 0.57568359375, -0.00844573974609375, 1.69921875, -0.49267578125, -0.1561279296875, 1.3486328125, -1.7265625, 0.1400146484375, 0.11669921875, -0.83251953125, -0.03704833984375, -1.0244140625, -1.1796875, 0.2012939453125, -0.0970458984375, 1.6015625, -0.87646484375, -1.177734375, -0.736328125, 2.12109375, 0.85009765625, 0.048187255859375, -1.4091796875, -0.79296875, -0.986328125, -0.70068359375, 0.0110931396484375, 0.2449951171875, 0.77001953125, -0.66796875, -0.09442138671875, 0.459228515625, 0.93896484375, 0.72265625, -1.5146484375, -0.5517578125, 2.3984375, -0.33251953125, -0.74365234375, -0.25048828125, 0.84033203125, 0.3193359375, 0.7138671875, 0.88720703125, -1.134765625, -0.75537109375, 0.9736328125, 1.3193359375, -0.363037109375, -0.420166015625, -0.1046142578125, 1.79296875, 0.11907958984375, -0.66162109375, -0.68798828125, 0.48974609375, 0.80419921875, 0.259765625, -0.908203125, -0.428466796875, 0.43212890625, -0.8134765625, 1.41015625, -0.86279296875, 0.578125, 0.216796875, 0.89892578125, -0.039886474609375, 1.0341796875, 0.95654296875, 0.55810546875, -0.25244140625, -0.5576171875, 1.4970703125, -1.9814453125, -1.0791015625, 0.47412109375, -0.5263671875, 1.455078125, -1.490234375, -1.2666015625, -0.60107421875, -1.3515625, 1.3232421875, -1.3505859375, 0.10675048828125, 0.398193359375, 1.0908203125, 1.072265625, -2.10546875, 0.35498046875, 0.339599609375, -0.302001953125, 0.322021484375, 2.3359375, -1.181640625, 0.7216796875, 0.50146484375, -0.08587646484375, -0.78662109375, -1.1181640625, 0.445556640625, -1.4287109375, -1.2509765625, -0.26123046875, 0.2120361328125, 0.45166015625, -0.300537109375, 1.1005859375, -2.07421875, -1.08203125, -0.2998046875, 1.7216796875, 0.3310546875, -2.2421875, 0.9208984375, -0.283203125, -0.3984375, -0.0205535888671875, -1.3740234375, -0.91845703125, -0.2216796875, 1.44921875, -1.3408203125, -0.1981201171875, 0.11114501953125, -0.53857421875, 1.2080078125, 1.134765625, -0.1348876953125, 0.80126953125, 0.128662109375, -0.89111328125, -0.457763671875, 0.8466796875, 1.8466796875, 0.97216796875, 0.572265625, 0.0787353515625, -0.07757568359375, 1.9130859375, 0.1859130859375, 0.1824951171875, 0.62841796875, 0.313232421875, -0.328125, -1.3671875, 1.041015625, -0.390380859375, 0.60498046875, 0.814453125, -1.759765625, -1.9267578125, -1.4306640625, 1.0615234375, -1.2119140625, 1.1533203125, -0.18603515625, -0.5888671875, 0.93603515625, -0.2254638671875, -0.8984375, 0.307861328125, -0.6533203125, -2.306640625, 1.6513671875, -1.4189453125, -0.43408203125, -0.4111328125, -0.79638671875, 0.30615234375, 1.6484375, 0.26953125, -0.03857421875, 0.44287109375, -1.4423828125, -0.391357421875, -1.0986328125, -0.11480712890625, -0.421875, -1.1162109375, 0.46533203125, 0.0148468017578125, -0.87890625, 0.701171875, -2.8828125, 0.9072265625, -1.783203125, 0.9912109375, 1.0068359375, -0.7607421875, -0.53466796875, 0.08331298828125, -1.138671875, 1.6201171875, -0.0224151611328125, -1.7314453125, 0.748046875, 0.98388671875, -0.99365234375, 0.65185546875, -0.37646484375, -1.5087890625, 1.021484375, -0.41845703125, -0.368896484375, -0.1744384765625, 0.423583984375, -0.9296875, -1.3115234375, 0.60107421875, -0.60107421875, 0.6357421875, -0.2119140625, -1.3876953125, -0.79931640625, -1.43359375, 0.66015625, 0.311767578125, 0.56494140625, -0.1539306640625, -0.01406097412109375, 1.0537109375, 0.12249755859375, -0.01004791259765625, -0.46484375, -0.794921875, 0.611328125, 0.0258941650390625, 0.26611328125, -1.216796875, 0.0155029296875, -1.3671875, -0.42919921875, -0.7412109375, -0.0212554931640625, 0.7431640625, 1.013671875, -2.39453125, 1.1552734375, -2.28125, -0.0333251953125, 1.890625, 0.15234375, 1.2822265625, -0.5849609375, 1.193359375, 0.88427734375, -0.93505859375, -0.0345458984375, 2.619140625, 0.82275390625, -0.9755859375, 0.27001953125, 0.53466796875, 0.10662841796875, 1.44140625, -0.79931640625, -1.6640625, 0.494384765625, -0.132568359375, 0.8173828125, -0.31689453125, 0.280517578125, -0.29931640625, -0.7109375, 0.0810546875, -0.4052734375, 1.4931640625, -0.08050537109375, -2.2890625, -1.0673828125, -2.103515625, -0.051300048828125, 1.6357421875, 0.6875, -0.51806640625, 0.375, -1.3828125, -1.703125, -1.0029296875, -0.3974609375, -0.1055908203125, 0.002780914306640625, 0.8740234375, -0.384521484375, 0.09771728515625, -0.6591796875, -0.66064453125, 0.3583984375, -1.173828125, -0.9287109375, -0.271728515625, 0.08062744140625, 0.943359375, -0.63671875, -0.94921875, -1.9072265625, -1.3974609375, 0.95849609375, -1.841796875, -1.474609375, 1.0537109375, -1.3046875, -2.359375, 0.64111328125, -0.36865234375, -3.033203125, 2.30078125, 0.47412109375, -0.5869140625, 0.3671875, 1.4951171875, -0.51904296875, -0.414794921875, 0.76806640625, -0.75146484375, -0.61181640625, -0.85009765625, -0.1912841796875, 0.17578125, -0.313232421875, -0.10760498046875, -1.13671875, 1.181640625, -1.041015625, 1.115234375, 1.19921875, -1.361328125, 2.201171875, -0.445068359375, -1.591796875, -0.33935546875, 1.3525390625, 1.0244140625, 0.73876953125, -0.7158203125, -0.65576171875, 0.5546875, 0.1656494140625, 1.748046875, -2.048828125, -0.478271484375, -1.6181640625, -0.2296142578125, -1.427734375, -1.015625, 0.1328125, 0.60107421875, 1.341796875, 2.197265625, 0.173095703125, 0.94677734375, 0.173095703125, 0.99462890625, -0.96435546875, -0.07763671875, 0.43798828125, -0.397216796875, -0.85400390625, -0.1441650390625, -1.56640625, -0.195556640625, 0.37890625, -3.22265625, 0.08123779296875, 1.3857421875, 0.052215576171875, 0.44189453125, 0.35302734375, -0.6533203125, -1.3955078125, 0.1405029296875, 1.6611328125, -1.0322265625, -2.197265625, 1.8212890625, 0.6259765625, 0.25048828125, -2.30859375, -0.219482421875, -1.19140625, -0.424072265625, 0.29638671875, -1.865234375, -0.802734375, 0.87890625, -0.6044921875, 0.578125, -0.669921875, 0.2403564453125, 0.99560546875, -0.1937255859375, -1.2177734375, -1.5009765625, -0.231201171875, -0.3642578125, 0.31201171875, 1.7041015625, -0.49658203125, 0.67236328125, 0.369873046875, 0.70703125, -2.611328125, 0.63671875, 2.94921875, -1.130859375, 0.205810546875, 0.136474609375, 0.06585693359375, -0.368408203125, -1.189453125, 1.345703125, 0.315185546875, -0.1240234375, -0.77783203125, -1.287109375, 0.4345703125, -1.4228515625, -0.10565185546875, 0.280517578125, -0.58349609375, 0.7265625, 0.9921875, 0.1756591796875, 0.1815185546875, 0.1533203125, 0.54833984375, -1.1103515625, 1.0234375, -0.1517333984375, 1.134765625, -0.95751953125, 1.111328125, -0.13330078125, 1.712890625, -0.06103515625, 0.6806640625, 0.1142578125, 2.068359375, -1.0615234375, -0.499267578125, 0.5556640625, -0.386474609375, -1.130859375, 0.80224609375, 0.4912109375, 0.875, -1.5546875, -0.28369140625, 0.2188720703125, 0.57421875, 0.50927734375, -0.2442626953125, 1.0869140625, 1.337890625, 0.41748046875, -2.078125, -1.5068359375, -0.31005859375, 0.10552978515625, -1.2685546875, 0.03399658203125, -0.2017822265625, -0.2442626953125, 0.6787109375, -0.64111328125, -1.1806640625, 0.279541015625, 0.0299224853515625, -0.15771484375, 0.79345703125, -1.6572265625, -1.330078125, 0.8603515625, -0.01861572265625, -1.5009765625, 0.131591796875, 0.3076171875, -0.72607421875, -0.050872802734375, -1.3125, 0.3525390625, 0.7763671875, -0.65966796875, 0.62109375, 1.2138671875, 1.7373046875, -1.5625, -0.84814453125, -1.2236328125, -0.60205078125, 0.85693359375, -0.36474609375, -1.95703125, 0.1844482421875, -0.63623046875, 0.28125, 0.67529296875, -0.039031982421875, -0.1260986328125, 0.82666015625, 0.1490478515625, -0.13134765625, -2.421875, -0.3046875, -0.72021484375, 0.67529296875, 0.76806640625, -0.299560546875, -0.2225341796875, 0.96435546875, 0.99365234375, -0.0318603515625, -1.0185546875, -0.69189453125, -0.447998046875, -0.262939453125, -0.469482421875, -1.478515625, -1.6171875, -0.40283203125, -1.31640625, 0.7421875, 0.0278472900390625, -0.6484375, 0.438720703125, 0.1961669921875, 0.63671875, -0.55615234375, -0.81201171875, -0.7861328125, 1.8564453125, -0.1934814453125, -0.0794677734375, 0.14453125, -0.5146484375, 0.11053466796875, -0.058013916015625, 0.86328125, -1.8701171875, -1.71484375, -0.1744384765625, -0.10931396484375, -1.017578125, 1.326171875, 0.8681640625, -0.10626220703125, 0.556640625, 2.251953125, -0.415771484375, -0.32080078125, 0.578125, 0.1129150390625, 0.423583984375, 1.6884765625, 0.08721923828125, -0.376708984375, -1.125, -1.185546875, 0.0606689453125, 0.912109375, -0.27490234375, 0.1761474609375, -0.1240234375, -1.412109375, -1.4189453125, -1.43359375, 1.6162109375, -0.1551513671875, -0.68505859375, -0.2093505859375, 0.30712890625, 1.234375, -0.100341796875, 0.3701171875, 0.328369140625, -0.68603515625, -0.442626953125, 0.364990234375, 1.9296875, 2.185546875, 1.1669921875, -0.77587890625, 0.591796875, 1.248046875, 0.0160675048828125, -1.1181640625, 0.498291015625, -1.6064453125, 0.5859375, -0.65869140625, -0.4287109375, -0.258544921875, -0.400634765625, -0.312255859375, -0.39111328125, -0.2388916015625, -1.0478515625, 1.3251953125, -0.2626953125, 0.55859375, 1.11328125, 0.736328125, 0.480224609375, -1.5859375, -0.402587890625, 1.357421875, -0.0252838134765625, -0.498779296875, 0.27783203125, -0.82080078125, -0.6884765625, 0.72607421875, 0.755859375, 1.29296875, 0.59814453125, 1.5615234375, -1.244140625, 0.40478515625, -0.365966796875, 2.083984375, -0.7783203125, -2.25390625, 0.5537109375, -0.0106048583984375, -0.3115234375, -0.31640625, -0.022705078125, 1.9912109375, -0.55224609375, -0.56298828125, 0.303466796875, -1.435546875, 0.79443359375, -0.1478271484375, 1.537109375, -0.85009765625, -0.73876953125, -0.91943359375, -1.3701171875, -0.323974609375, 0.9794921875, 0.1868896484375, -0.298095703125, 0.80078125, -1.6337890625, 0.12060546875, 0.456787109375, 0.355224609375, -1.6044921875, -0.162353515625, 0.751953125, -1.556640625, 1.3095703125, 0.2509765625, 0.6552734375, 0.3017578125, 0.52001953125, 1.791015625, 1.8388671875, 0.6328125, -0.485107421875, -1.82421875, 2.26953125, 0.377685546875, -0.059478759765625, 0.354736328125, -0.7353515625, 0.3447265625, 0.61279296875, -1.3095703125, 0.3525390625, 0.83056640625, 0.39013671875, 0.580078125, -1.2431640625, 0.11053466796875, 0.1177978515625, -1.08203125, 0.6943359375, -0.28515625, 0.212646484375, 1.15234375, 1.0908203125, -0.222412109375, -1.958984375, -0.291748046875, 0.2342529296875, 0.297119140625, 0.50634765625, -0.40966796875, 0.234375, -0.1668701171875, -1.1748046875, -1.744140625, -0.83837890625, -0.6435546875, 0.244873046875, -1.1181640625, 0.720703125, 1.5888671875, -0.1676025390625, 0.319091796875, -2.095703125, 1.5712890625, -0.9345703125, -0.68798828125, -0.22607421875, 0.89990234375, 0.71826171875, 0.67041015625, -0.8388671875, -0.265869140625, 1.111328125, 1.138671875, -0.521484375, -0.5283203125, -2.271484375, -0.42578125, -0.66650390625, -0.0587158203125, -0.035491943359375, 0.58154296875, 1.306640625, -2.8203125, 0.055572509765625, -0.09893798828125, 0.027130126953125, 1.4814453125, -1.1533203125, -0.003520965576171875, -0.4521484375, -2.509765625, -0.6357421875, -0.1783447265625, 0.05218505859375, 1.841796875, -0.9541015625, -0.435546875, 1.3720703125, 0.25439453125, 1.158203125, 0.411865234375, -0.1722412109375, -0.630859375, 1.607421875, 0.77734375, -0.080322265625, -0.2459716796875, -0.0643310546875, -0.66259765625, 1.3115234375, -0.192138671875, 0.63916015625, -1.92578125, -0.6923828125, 0.220458984375, 0.0823974609375, 0.64990234375, 0.0968017578125, -1.7568359375, -0.09393310546875, 1.015625, -0.2457275390625, -0.0706787109375, -1.19921875, -0.8369140625, -0.64794921875, -1.6806640625, -0.44921875, 0.595703125, 0.225830078125, -1.1953125, -1.2529296875, 1.818359375, -1.1259765625, 0.39892578125, -0.86474609375, -0.1514892578125, 1.2021484375, -1.1298828125, -0.323974609375, 1.0224609375, -0.53662109375, 1.810546875, -0.8095703125, 0.122314453125, 0.0181427001953125, -0.73583984375, -1.5, -1.1904296875, 0.06256103515625, 1.0908203125, -1.4296875, 0.140869140625, -0.80126953125, -0.89599609375, 0.482177734375, 0.38330078125, 1.3310546875, 1.0625, -1.2158203125, -0.28564453125, 1.2138671875, -0.6064453125, -1.4619140625, -0.83740234375, 0.2734375, -0.46923828125, 0.2431640625, -0.85791015625, 1.615234375, 0.87939453125, -0.521484375, -0.71826171875, 0.91357421875, 0.662109375, -0.378662109375, -0.425048828125, 1.01171875, 0.849609375, 1.5244140625, 0.27685546875, -1.30859375, -0.36181640625, 0.00724029541015625, 1.7763671875, -0.124755859375, 0.6103515625, 1.67578125, -1.025390625, 1.341796875, 0.66552734375, -1.2509765625, 0.307373046875, 0.01062774658203125, -0.431884765625, -0.250732421875, 1.5634765625, 0.1031494140625, 0.59521484375, 0.2396240234375, -0.55322265625, 0.00983428955078125, -0.39111328125, 0.96923828125, 1.5908203125, -1.759765625, 0.8876953125, 0.0775146484375, 0.406494140625, 0.2149658203125, 0.359375, -0.80859375, 0.822265625, 0.837890625, -2.5546875, 0.8974609375, 0.77294921875, -0.470947265625, 0.65283203125, -0.060272216796875, -1.34375, 0.0806884765625, -0.673828125, 0.39697265625, -0.444091796875, -1.3271484375, 0.71435546875, -0.458984375, -0.08203125, 1.1357421875, -0.1619873046875, 0.2476806640625, 0.6103515625, 0.369873046875, -1.3427734375, -3.359375, 1.0693359375, -0.081787109375, 0.72705078125, 1.806640625, -0.424560546875, -0.1717529296875, -1.3232421875, 0.623046875, -0.6982421875, -1.2255859375, 0.237548828125, -0.2071533203125, 0.253173828125, 0.9365234375, 1.6572265625, 0.28515625, -1.0068359375, -0.457763671875, -0.52099609375, 1.673828125, 0.8603515625, -0.7646484375, -0.08935546875, -0.1597900390625, -0.390380859375, -0.07177734375, -1.103515625, -0.89990234375, 0.53955078125, 0.323974609375, 0.1717529296875, 0.09063720703125, 1.4228515625, 1.81640625, 0.69921875, 0.480712890625, 1.2060546875, -0.348876953125, -0.298095703125, -1.951171875, -0.69970703125, 0.03363037109375, 0.47509765625, -0.453125, 0.454833984375, -0.035797119140625, 0.705078125, 0.4677734375, -0.8505859375, 0.59326171875, -0.57275390625, -0.75537109375, -2.005859375, 0.822265625, 1.8310546875, 0.0548095703125, 0.53857421875, -1.1953125, 0.8408203125, -0.517578125, -0.1612548828125, -0.75537109375, 0.80615234375, 0.3330078125, -1.1689453125, -0.3359375, 0.62255859375, 0.1798095703125, -0.2076416015625, -1.7421875, 0.0850830078125, 1.6435546875, 0.7646484375, -0.020416259765625, -0.68359375, -0.619140625, 0.030364990234375, -0.408447265625, -0.33935546875, 0.8330078125, 0.5380859375, 0.474609375, 0.0298614501953125, -0.85595703125, 0.33642578125, -0.1673583984375, -0.1668701171875, 0.1728515625, -1.349609375, -2.28125, 0.2142333984375, -1.0087890625, 1.3759765625, 0.2210693359375, -0.7919921875, -0.94873046875, 0.1795654296875, -1.7275390625, 1.8310546875, 1.923828125, -0.65234375, -0.892578125, 0.6904296875, -0.388427734375, -0.1734619140625, 0.391845703125, -0.189697265625, 1.01171875, -2.12109375, 0.802734375, -0.096923828125, -0.44140625, -1.0224609375, -0.264892578125, 1.4267578125, -0.26171875, 0.89111328125, -0.179931640625, 2.12109375, -0.82861328125, -0.489013671875, 1.07421875, 0.5771484375, 0.25439453125, -1.5107421875, 1.591796875, -1.3349609375, 0.970703125, 0.0621337890625, 2.255859375, -0.0008177757263183594, -0.388671875, 0.481201171875, 1.1103515625, 0.85595703125, -0.1229248046875, -0.07562255859375, 0.1961669921875, 0.41748046875, 1.6962890625, 0.720703125, -0.077880859375, -0.86669921875, 0.52734375, 1.2353515625, -1.09375, 0.51904296875, 0.38427734375, 0.2587890625, -1.89453125, 1.818359375, 0.35546875, 0.64599609375, 0.7431640625, -0.58154296875, -1.234375, -0.6630859375, 0.7373046875, -0.38330078125, 0.32861328125, -0.447265625, -0.255126953125, 1.6123046875, -1.8720703125, -0.457763671875, -0.61083984375, 0.59375, 0.0212860107421875, 0.71826171875, 0.1083984375, 0.1375732421875, -2.390625, 0.46728515625, 1.1513671875, -0.20654296875, -0.990234375, 1.587890625, -0.489990234375, -1.65625, -1.580078125, -0.6416015625, 0.6162109375, -0.9462890625, 0.278564453125, -0.2410888671875, -0.07781982421875, -0.312255859375, -1.466796875, 0.01296234130859375, -1.2236328125, 1.103515625, 1.0087890625, -2.4140625, -0.121337890625, 0.58740234375, 1.4423828125, -0.37353515625, -0.51806640625, 1.3212890625, -0.0150909423828125, -0.998046875, -0.8017578125, -0.728515625, -1.80078125, -0.004177093505859375, -0.291015625, 0.352294921875, -0.37939453125, 0.5712890625, 0.092041015625, 1.0771484375, -0.5126953125, -1.462890625, 1.90625, 1.6669921875, -0.74658203125, -0.317626953125, 0.1680908203125, -2.01171875, -0.245849609375, -0.22314453125, -0.138671875, 1.3095703125, -0.294677734375, -1.173828125, -0.049774169921875, 2.44921875, 0.253173828125, 0.34521484375, -1.109375, 0.1964111328125, 0.4296875, 2.1328125, -1.73046875, 1.4892578125, -0.2198486328125, -1.1123046875, -0.62158203125, 2.970703125, 0.66796875, -1.3759765625, -0.21337890625, 0.1734619140625, -1.3046875, 0.95947265625, -0.30322265625, -0.6044921875, 0.1510009765625, 0.54931640625, -0.65380859375, -1.154296875, 2.736328125, 0.110107421875, 0.767578125, -0.89599609375, 0.615234375, -1.1328125, -1.2568359375, 0.439453125, -0.96826171875, 0.303466796875, 0.8701171875, 0.421142578125, -0.486328125, 0.4501953125, 0.2783203125, 0.314453125, -0.6337890625, 0.489990234375, -2.03125, -1.486328125, -0.6083984375, 0.69921875, -1.603515625, -0.28759765625, -0.495361328125, -0.6552734375, 0.59765625, -1.310546875, -0.86376953125, -1.5673828125, -0.833984375, -0.65771484375, -0.320068359375, -0.49267578125, 0.2474365234375, 1.7900390625, 1.517578125, 1.0546875, 1.1171875, -1.7919921875, -0.001251220703125, -0.06854248046875, 0.94873046875, 0.58837890625, -0.2147216796875, 1.4521484375, -2.552734375, 0.29150390625, 0.84521484375, -1.1005859375, -0.85302734375, -0.50830078125, -0.4091796875, 1.0673828125, 1.28515625, -0.818359375, 1.154296875, -0.13720703125, 0.2060546875, 0.430908203125, -0.5361328125, -0.227783203125, 0.97265625, -0.173095703125, 1.9248046875, -0.056640625, 0.4912109375, -0.385498046875, 0.255615234375, -0.005451202392578125, 0.1781005859375, -0.0305938720703125, 1.3935546875, 1.482421875, -0.2626953125, 0.1827392578125, -0.98779296875, 2.228515625, -0.6220703125, 0.76123046875, 0.380615234375, -0.55029296875, -0.71923828125, -0.8916015625, -0.7109375, 0.7431640625, 1.525390625, 1.2412109375, 0.05279541015625, -0.26220703125, 0.958984375, -1.0234375, 0.00016987323760986328, 1.3251953125, 1.4775390625, 1.30078125, 0.50830078125, -0.1317138671875, -0.28564453125, -0.708984375, -0.71484375, 0.1273193359375, -0.65869140625, -0.0628662109375, -0.912109375, -0.74951171875, -0.55810546875, 2.138671875, -0.261474609375, -1.064453125, -0.71826171875, -1.619140625, -1.2109375, 1.3505859375, 0.7607421875, -0.99365234375, -1.296875, -0.83447265625, 1.5625, -0.57958984375, 1.033203125, 0.68701171875, -0.1883544921875, -2.1015625, -0.712890625, 0.408447265625, 0.68359375, -1.1435546875, -0.17138671875, 0.89697265625, 0.75390625, 0.1409912109375, -0.07574462890625, -0.52587890625, -0.35986328125, 1.2734375, 0.72705078125, 0.984375, 1.0927734375, 0.03692626953125, -0.10736083984375, -1.4384765625, -0.2188720703125, 1.6435546875, 0.0985107421875, 0.57958984375, -0.48828125, 0.1846923828125, -0.329833984375, -0.322509765625, -0.7109375, 0.89208984375, 0.91455078125, -0.14013671875, -1.037109375, 0.2103271484375, -0.11785888671875, 0.63916015625, -0.6318359375, -0.429443359375, 0.61376953125, 1.24609375, -1.3701171875, -0.79345703125, 0.1497802734375, 0.775390625, -1.53125, -1.203125, 0.2763671875, 1.48828125, -0.1478271484375, -1.2890625, -0.391845703125, -0.265869140625, -0.223388671875, 2.22265625, 0.9208984375, -0.59423828125, 0.298095703125, 3.056640625, -1.623046875, 0.9228515625, -2.80078125, -0.034332275390625, -1.037109375, 0.054473876953125, 0.53125, -0.09295654296875, -0.08343505859375, 0.6396484375, -1.08203125, -1.04296875, 1.755859375, 0.58740234375, -0.5478515625, 0.77392578125, -1.064453125, -0.62890625, 0.5322265625, -1.498046875, -0.1246337890625, 0.58837890625, -2.044921875, 0.7998046875, -0.1024169921875, 0.89453125, -1.193359375, 0.66796875, 0.414306640625, -0.79736328125, -2.38671875, -0.505859375, 0.488525390625, 0.438720703125, -1.5888671875, -0.388427734375, 0.412353515625, 0.284423828125, -0.034912109375, -0.462158203125, -0.263916015625, -0.62744140625, -1.361328125, 0.49658203125, 0.81591796875, 0.81591796875, -1.541015625, 1.1484375, -3.318359375, 2.091796875, 0.34912109375, 1.4052734375, -0.2822265625, -0.24462890625, 0.01342010498046875, 0.393798828125, 1.7392578125, 0.6318359375, -1.61328125, 0.84423828125, 1.4677734375, -1.0625, 0.04388427734375, -0.78515625, 0.1881103515625, -0.2413330078125, 0.5302734375, 0.89892578125, 1.744140625, -0.8876953125, -1.234375, 0.791015625, 1.818359375, -0.65673828125, -2.556640625, -0.53076171875, 1.4033203125, 0.90478515625, -0.08941650390625, -1.533203125, 1.7060546875, 0.193359375, 1.2177734375, 1.2490234375, -1.91015625, -1.48828125, 0.8642578125, 0.96630859375, -0.875, -0.84033203125, -2.6640625, -0.58740234375, -0.107421875, 0.48291015625, -0.80224609375, -0.6025390625, -2.140625, -0.68701171875, -1.1845703125, 1.41796875, 1.009765625, 1.255859375, 0.3984375, 1.142578125, -0.07794189453125, -0.5693359375, -0.267578125, 0.8349609375, -0.51904296875, 0.7490234375, 0.20068359375, 1.5166015625, -1.4541015625, 0.53076171875, 0.91796875, 1.505859375, -1.099609375, -0.06695556640625, -0.60302734375, -1.32421875, -0.4169921875, -2.146484375, 0.60205078125, -0.406494140625, -0.662109375, -0.173828125, -1.5263671875, 2.0078125, 1.54296875, 1.4619140625, 1.2978515625, 0.2293701171875, -1.43359375, 0.91162109375, 0.276611328125, -1.193359375, 0.416259765625, -0.0594482421875, 1.9013671875, -2.1796875, 1.8017578125, 0.74169921875, -0.50634765625, -0.93701171875, 1.5654296875, -0.298583984375, 1.0654296875, 0.53857421875, 0.50537109375, 1.4140625, -0.377685546875, -0.12255859375, 1.607421875, -1.966796875, -0.9521484375, 1.203125, 0.76416015625, 0.0419921875, -0.2257080078125, 0.205322265625, -1.1357421875, -0.8740234375, -0.312744140625, -0.301025390625, -0.94677734375, 1.1435546875, 1.01171875, -1.0791015625, 0.79638671875, 1.28515625, -0.9921875, 0.416748046875, -0.4140625, 0.2459716796875, -1.6005859375, -0.98095703125, -0.65576171875, -0.107421875, -0.201416015625, -0.6162109375, 0.88525390625, 0.1837158203125, -1.0087890625, -1.521484375, -0.410888671875, -1.013671875, 1.9912109375, 1.408203125, 0.3095703125, -0.370361328125, 1.6943359375, 1.6201171875, 1.7783203125, -1.095703125, 0.6982421875, 0.4697265625, 1.1025390625, -1.7333984375, 1.17578125, -0.50146484375, -1.69921875, 0.85693359375, 1.1435546875, -0.15966796875, 0.3623046875, 0.1767578125, 1.6962890625, -0.0192413330078125, 0.2003173828125, -0.83837890625, 0.2288818359375, 1.716796875, 1.66796875, 0.09027099609375, 1.9267578125, -0.6103515625, 1.583984375, 1.8828125, -1.1005859375, 0.82373046875, 0.285400390625, 0.2330322265625, 1.0654296875, 0.210205078125, 1.494140625, -0.479736328125, 0.46337890625, -1.015625, 0.64990234375, -0.58251953125, -0.86279296875, 0.1351318359375, 0.85205078125, 1.8310546875, -0.345703125, 0.251220703125, 0.2132568359375, 0.186767578125, -0.485107421875, -0.482666015625, -1.2353515625, 0.0924072265625, 0.11309814453125, 0.10479736328125, 2.044921875, 0.88818359375, -1.251953125, -0.7802734375, -0.50146484375, 0.77490234375, -0.91845703125, -0.53466796875, 0.30078125, 1.7392578125, -0.60009765625, 0.464599609375, -1.1025390625, 0.0855712890625, 0.303955078125, 0.392578125, -0.1390380859375, 0.7216796875, -1.4873046875, 0.1314697265625, -1.6689453125, 0.1978759765625, 0.63818359375, 0.73876953125, 1.2880859375, -1.666015625, -0.3466796875, 0.7138671875, 1.375, -0.2490234375, 0.022216796875, 0.58642578125, -0.0477294921875, -0.1805419921875, -0.08154296875, -1.4580078125, -0.062744140625, -2.1875, -1.05859375, -0.2493896484375, -0.231201171875, 0.4892578125, 0.9326171875, 0.97802734375, -0.458251953125, -0.208984375, 0.52294921875, 0.92626953125, 0.1282958984375, -0.68603515625, 0.058197021484375, 0.81640625, -0.453125, 1.3623046875, 0.464111328125, -0.8798828125, 1.9111328125, 0.222412109375, 0.1126708984375, 1.4267578125, -1.119140625, 1.4140625, 1.083984375, -0.024627685546875, -0.72021484375, 1.259765625, -0.44140625, -1.23046875, 0.54150390625, 1.3759765625, -1.30078125, 0.37939453125, 0.369384765625, 2.12109375, -0.5322265625, 0.576171875, 0.0771484375, 0.0236663818359375, 1.529296875, 0.78076171875, -0.67236328125, -2.279296875, -1.65234375, -1.619140625, -0.537109375, 0.60595703125, -0.83203125, -1.2939453125, -0.2626953125, 0.361328125, 0.2108154296875, 0.40869140625, -1.6279296875, 0.77001953125, -0.623046875, 1.3662109375, -0.95947265625, 0.06927490234375, 0.428955078125, -0.2100830078125, 1.7216796875, 1.3486328125, 0.93359375, 0.1326904296875, 0.2247314453125, -0.271728515625, -0.4853515625, -0.31103515625, -0.4013671875, 0.1688232421875, 1.001953125, -0.71044921875, -1.55078125, 0.2890625, -1.421875, -0.833984375, -1.115234375, -0.3154296875, 0.9140625, -0.97216796875, -0.323486328125, 0.56298828125, 1.7568359375, -0.39697265625, -0.75439453125, -0.1295166015625, 2.33984375, 1.861328125, -0.94580078125, 1.419921875, 0.1097412109375, -0.3173828125, 0.1986083984375, -1.5654296875, 0.310546875, -0.01410675048828125, 0.423828125, 0.62841796875, 1.93359375, 2.166015625, 0.51025390625, -1.171875, -0.90478515625, -0.341064453125, -1.78125, 0.9404296875, -1.609375, 0.07763671875, -1.333984375, -0.81982421875, 0.59716796875, -0.09503173828125, 1.0888671875, -0.6416015625, 1.6513671875, -0.70751953125, -0.9931640625, -1.6162109375, -1.115234375, -1.474609375, -0.11865234375, 0.8115234375, 0.287109375, -0.91064453125, 0.52197265625, 0.21142578125, 0.724609375, -0.229736328125, 0.40869140625, -0.99951171875, 1.0556640625, 0.0814208984375, 1.337890625, 0.76171875, 0.06866455078125, -1.2802734375, -0.036712646484375, 0.16552734375, 0.038604736328125, -0.166259765625, 1.2080078125, 0.54296875, 1.2109375, -0.85986328125, -1.228515625, -1.41796875, 0.06634521484375, 1.060546875, -0.8212890625, -1.3310546875, -0.94091796875, -0.55810546875, -0.8408203125, -0.93115234375, -0.375244140625, -0.8681640625, -1.6435546875, -0.9228515625, 1.3212890625, -1.40625, 0.9365234375, 1.0146484375, 2.26171875, -0.277587890625, -1.416015625, -0.50341796875, -1.8515625, 0.96484375, 0.279541015625, 1.8623046875, 0.83447265625, 0.79736328125, 0.251708984375, -1.8076171875, -0.0015192031860351562, 0.939453125, -1.6611328125, -0.71728515625, -1.2373046875, -0.61376953125, -0.421142578125, -0.1414794921875, -0.14697265625, -1.3505859375, -2.427734375, 0.137939453125, 0.36865234375, 1.1044921875, -0.58203125, 1.169921875, 1.3173828125, -1.26953125, -0.251708984375, 0.94189453125, -0.46630859375, 0.405029296875, -0.2705078125, 0.470458984375, 0.79345703125, -0.6552734375, -0.2257080078125, 0.34375, 0.2115478515625, 0.40966796875, -1.0859375, -0.1845703125, -1.380859375, -1.3134765625, -0.289306640625, 0.405517578125, -0.6533203125, 0.55419921875, -0.1759033203125, 0.94677734375, -0.02130126953125, -1.30078125, 0.12457275390625, 1.41015625, -0.5205078125, 0.4990234375, 1.673828125, -1.1083984375, -0.94091796875, -0.87841796875, -1.341796875, -0.84619140625, 0.73291015625, 0.27734375, 0.8544921875, -0.33984375, -1.5390625, -1.677734375, -1.6337890625, -0.64794921875, 1.2685546875, 1.35546875, -0.063720703125, -0.398193359375, 0.322265625, 1.3173828125, -0.3486328125, -0.59228515625, 0.05657958984375, -0.86474609375, 0.66796875, 0.73779296875, 0.2435302734375, 0.59228515625, -0.027618408203125, -1.1162109375, -1.8291015625, -0.29150390625, -1.064453125, -0.29931640625, -0.018951416015625, 0.5634765625, -0.615234375, 0.7919921875, 1.8115234375, -1.2421875, 1.6474609375, 0.436767578125, -0.2353515625, 0.229248046875, -0.11785888671875, -0.211181640625, 0.0975341796875, 1.4931640625, 1.3212890625, -0.359375, -0.76611328125, -0.22705078125, -0.2734375, 0.343994140625, -0.347900390625, 0.548828125, -0.395751953125, 0.278076171875, 1.62109375, 1.0146484375, 0.982421875, -2.0546875, -2.15625, 0.1295166015625, 0.68359375, 1.53125, 0.75244140625, -0.35009765625, -0.55419921875, -1.712890625, -1.5283203125, -0.93359375, -1.2861328125, 0.476806640625, 0.11749267578125, -1.068359375, 2.5625, 0.736328125, -0.2071533203125, -1.861328125, -0.31298828125, 1.7431640625, -0.0301361083984375, -0.66748046875, -0.193359375, 0.93701171875, 0.71337890625, 0.98681640625, 1.9140625, 0.642578125, 1.46875, -0.12139892578125, -0.837890625, 0.2415771484375, 0.27099609375, -0.9521484375, -0.61669921875, -2.01171875, -1.216796875, 1.6806640625, 0.92724609375, -0.2308349609375, -0.473876953125, -1.916015625, 1.0517578125, 1.2255859375, 0.3203125, 1.7314453125, 0.6650390625, -0.6689453125, 0.1419677734375, -0.58935546875, -1.8623046875, -0.269287109375, 0.1854248046875, 1.7666015625, 0.990234375, 0.72119140625, 0.39990234375, -0.272705078125, -0.052032470703125, -0.5810546875, -0.30419921875, -1.359375, 0.775390625, -1.08203125, -0.1263427734375, 0.72705078125, -0.29150390625, -0.68701171875, 0.50341796875, 0.78125, 0.302490234375, 0.5810546875, 1.49609375, 1.11328125, -0.79931640625, -1.4716796875, -1.3798828125, -2.484375, 1.998046875, -1.701171875, -2.390625, -0.41748046875, 1.3134765625, 1.6611328125, -1.3056640625, -0.70263671875, -0.34716796875, 2.333984375, -0.19873046875, -2.287109375, -0.9638671875, 0.2451171875, -0.96435546875, 0.0325927734375, 1.9755859375, -0.67529296875, -1.6640625, 0.0823974609375, -1.7353515625, 1.12890625, 0.44091796875, 1.3203125, 1.8330078125, -0.1591796875, 1.5224609375, -0.79248046875, -0.1383056640625, 0.439697265625, 1.1005859375, 0.11456298828125, -0.9130859375, -0.0018110275268554688, -3.041015625, 0.2666015625, -0.642578125, 0.8291015625, -0.96435546875, 1.8857421875, 0.94091796875, -0.61572265625, -2.912109375, -1.150390625, -0.01520538330078125, -1.6064453125, 0.31982421875, 0.8759765625, -1.0341796875, 0.08935546875, 1.416015625, 0.6630859375, 0.491455078125, 0.15966796875, 0.338623046875, -1.3974609375, -0.7900390625, 0.68212890625, 1.3212890625, -0.36767578125, -0.317626953125, -0.82177734375, 1.43359375, -1.2158203125, 1.119140625, 1.130859375, 0.143310546875, -0.70703125, -0.8037109375, -0.60205078125, -2.15625, -0.346923828125, -0.85302734375, 0.7119140625, 0.11175537109375, -0.62646484375, -0.01708984375, -0.54443359375, -0.299560546875, -0.107666015625, -0.0609130859375, 0.454833984375, -1.0107421875, -0.48291015625, 1.865234375, -1.4716796875, -1.3212890625, -1.3798828125, 0.2327880859375, 0.3037109375, -0.2244873046875, -0.78466796875, -0.92138671875, 0.5673828125, -0.11920166015625, -0.26904296875, -1.8349609375, 0.837890625, -0.361083984375, -1.2802734375, -1.119140625, -0.1326904296875, 1.984375, 0.1630859375, -0.943359375, 2.091796875, 0.7021484375, -1.42578125, 0.305908203125, -2.109375, -1.126953125, -1.39453125, 1.5888671875, -0.9169921875, -1.0537109375, 0.7724609375, 0.748046875, 0.77197265625, -0.412109375, 0.884765625, -0.92529296875, -0.44970703125, -0.10772705078125, 1.0859375, -3.267578125, -0.246826171875, 0.31689453125, -0.54443359375, 1.787109375, 0.53515625, 0.405029296875, 0.203125, -1.6103515625, 1.4423828125, -0.71728515625, -0.41943359375, 0.385986328125, 0.5703125, 0.427001953125, 0.890625, -2.486328125, -1.025390625, 1.1533203125, -0.121337890625, -1.134765625, 0.1182861328125, -0.890625, 1.0654296875, -0.52978515625, -0.0223541259765625, -0.9375, 0.0257415771484375, -0.228271484375, -0.240234375, -2.6015625, -0.035125732421875, 0.73388671875, -0.716796875, 0.900390625, -0.1561279296875, 1.541015625, 0.177001953125, 0.841796875, -2.86328125, 0.026153564453125, -0.83349609375, -1.611328125, 0.00937652587890625, -0.1856689453125, 1.404296875, 0.60009765625, 0.2093505859375, -1.1220703125, 0.085693359375, -2.373046875, -1.7294921875, -1.1982421875, 0.986328125, 0.63037109375, 0.482421875, 1.2666015625, 0.4912109375, 0.404296875, 0.83642578125, -0.484619140625, -0.1295166015625, -0.1568603515625, 0.3154296875, -2.435546875, -2.32421875, 1.064453125, -0.931640625, 1.1279296875, -0.91796875, -0.74658203125, 0.89208984375, -2.23828125, -0.291748046875, 0.427978515625, 0.443603515625, 0.66796875, -1.7578125, -0.72900390625, 0.59130859375, 0.740234375, 1.08203125, 1.2265625, 0.01280975341796875, 1.447265625, 1.439453125, -0.46630859375, -0.466552734375, -1.205078125, 0.3095703125, -0.68115234375, 1.2451171875, -1.30078125, 0.91650390625, 1.03125, 0.51416015625, 1.5009765625, -0.69482421875, 0.78857421875, -1.5419921875, 1.4228515625, -0.358154296875, -0.2166748046875, -2.0625, -0.1385498046875, 0.1685791015625, 0.451416015625, -1.01171875, 1.7421875, 0.66162109375, 0.087890625, -0.83203125, -0.6044921875, 0.74609375, 2.41015625, -1.2880859375, -0.982421875, 0.289794921875, -0.5703125, 0.87744140625, -0.49462890625, 1.10546875, -0.52587890625, -1.048828125, 0.56396484375, -0.2276611328125, 0.350341796875, -1.8447265625, 0.708984375, 0.240478515625, 0.1741943359375, -1.8916015625, -0.6435546875, 0.07672119140625, -0.6572265625, -2.375, -0.50244140625, 0.4853515625, 1.1376953125, 0.5224609375, 1.3984375, -0.2021484375, -0.1572265625, 1.0810546875, 0.90673828125, 0.94384765625, -1.0068359375, 0.201416015625, 0.83837890625, 1.205078125, -0.5791015625, -0.068603515625, 0.37255859375, 1.474609375, 0.3212890625, 0.353271484375, -0.1497802734375, -0.77490234375, -0.96240234375, -0.90576171875, 1.70703125, -0.9833984375, 0.11083984375, -0.7587890625, 0.0836181640625, 0.97119140625, -0.088134765625, 0.185302734375, -0.66064453125, 0.19140625, -0.59765625, -1.3076171875, -0.1448974609375, 0.34716796875, 0.90576171875, -0.66357421875, -0.3447265625, 0.80322265625, 0.3193359375, 0.5869140625, -0.65869140625, -1.7890625, 1.4443359375, 0.1285400390625, -1.2822265625, 2.03125, -0.368896484375, 0.472412109375, 0.669921875, -1.1123046875, 0.0255126953125, -1.232421875, -0.431640625, -0.84326171875, -0.75390625, -1.23828125, 1.89453125, -0.0312042236328125, -0.2452392578125, 2.27734375, 0.92822265625, 1.34375, 0.1673583984375, 1.3193359375, 0.3837890625, 1.591796875, 0.75732421875, -1.595703125, -0.181884765625, 0.59423828125, -0.385986328125, 0.2841796875, -0.939453125, -1.0126953125, 0.267333984375, -1.1826171875, -1.25390625, -0.68115234375, -1.0791015625, -0.7099609375, -2.212890625, 0.80810546875, -0.69775390625, -0.91259765625, 0.1923828125, -2.095703125, 0.509765625, -0.296630859375, -0.5224609375, -0.6611328125, 0.70263671875, 2.09765625, 1.2158203125, 0.8564453125, -0.05963134765625, -0.970703125, 0.482666015625, -0.437744140625, -0.346923828125, 0.1866455078125, 0.59423828125, -0.5078125, 0.640625, -0.80859375, -0.05780029296875, -0.033233642578125, 0.150634765625, 0.41259765625, -0.327880859375, 0.074951171875, -0.47998046875, 0.2783203125, 1.33203125, -0.212646484375, 0.890625, -1.783203125, -0.127197265625, -0.53076171875, -1.2470703125, 1.185546875, 0.313720703125, -1.0654296875, 0.26953125, 1.9189453125, -0.37890625, -0.4248046875, -0.38427734375, 1.3818359375, 0.685546875, -0.86376953125, -0.0245819091796875, 1.3408203125, -0.2156982421875, -1.1044921875, -1.5068359375, -0.279296875, -0.58642578125, 1.6044921875, 0.43603515625, -1.580078125, 0.26318359375, 0.360595703125, -0.5517578125, 0.37939453125, -1.21484375, 0.93017578125, 1.208984375, 0.11981201171875, 0.108642578125, 0.8974609375, 0.1844482421875, -2.265625, 0.1129150390625, 0.43310546875, 0.306640625, 1.00390625, -0.701171875, 1.2646484375, 0.46533203125, 0.4443359375, -0.480712890625, 0.052459716796875, 0.356201171875, -0.11785888671875, 0.76708984375, 0.33544921875, 0.130859375, 0.8076171875, -0.4365234375, -0.00917816162109375, -0.416748046875, -2.6875, 0.54443359375, 0.69384765625, 0.263916015625, -0.0088958740234375, 0.158935546875, -0.8291015625, 0.06524658203125, 0.29736328125, 2.2421875, -1.021484375, 1.869140625, 0.53271484375, -0.327392578125, -1.0087890625, 0.9765625, 1.751953125, -0.728515625, 0.9677734375, 1.7451171875, 1.1240234375, -1.267578125, 0.99267578125, 0.26025390625, 0.53466796875, -0.1854248046875, -0.11279296875, 0.56982421875, 0.63916015625, -0.420166015625, 0.83642578125, -0.31005859375, -0.4775390625, -1.1953125, 0.0003063678741455078, -0.87158203125, -0.357421875, 0.8330078125, 1.0732421875, 2.673828125, 0.9833984375, -0.2362060546875, 0.78662109375, -1.25, -0.8857421875, -0.2242431640625, -1.0234375, -0.1036376953125, 0.290283203125, -1.1171875, 1.3603515625, -1.369140625, -1.783203125, 1.4228515625, 0.044586181640625, -0.9326171875, -0.1300048828125, 1.9833984375, -0.5166015625, 2.17578125, 1.1826171875, -1.076171875, -0.174560546875, 0.732421875, -0.08697509765625, 1.404296875, 1.6875, 0.36279296875, 0.062255859375, -2.201171875, 0.36767578125, -1.0986328125, 0.98388671875, -0.2186279296875, 3.26171875, -0.400634765625, -0.378662109375, -1.30078125, -1.560546875, -1.40234375, 1.392578125, 0.5908203125, 0.08984375, -1.7431640625, -0.73486328125, 0.498291015625, -0.1485595703125, -1.3330078125, 0.15771484375, -0.471435546875, -0.5576171875, -0.0061492919921875, -0.162841796875, 0.09161376953125, -1.0634765625, -0.18017578125, 2.00390625, -0.330322265625, 1.16015625, 0.498291015625, 0.52490234375, 0.05731201171875, 1.404296875, -0.253173828125, 0.131591796875, -1.9423828125, 1.2470703125, -1.712890625, -2.30078125, 2.08203125, 0.170654296875, 0.4609375, 0.1151123046875, 1.046875, 0.9658203125, -1.31640625, -1.9775390625, -1.330078125, -0.281494140625, -1.2119140625, 0.76953125, -0.61279296875, -0.95458984375, -0.421630859375, 0.7607421875, 1.44921875, -1.3525390625, 0.90234375, 0.5732421875, 2.638671875, -0.73583984375, 1.810546875, -1.9580078125, -0.18505859375, 0.347412109375, 0.5595703125, 0.89208984375, 0.13330078125, 0.360595703125, -0.6357421875, 0.2427978515625, 1.8955078125, -0.447998046875, -0.450439453125, -0.2001953125, -0.63037109375, 1.0009765625, -0.833984375, -0.304443359375, 0.2471923828125, 1.07421875, 1.4833984375, 4.0703125, -2.52734375, -1.0927734375, 1.1494140625, -0.357421875, -0.146240234375, -0.465576171875, 0.3720703125, -0.1405029296875, 0.86572265625, 0.44580078125, -0.1724853515625, -0.68212890625, -1.591796875, 0.7958984375, -0.1156005859375, 0.9892578125, 0.66162109375, 0.84716796875, 2.41015625, 0.5166015625, -1.7978515625, 0.67138671875, 1.4375, -2.294921875, -0.2471923828125, 0.85986328125, -2.189453125, -0.022491455078125, -0.1795654296875, 0.68994140625, 0.50048828125, 0.357421875, 0.390625, 0.8173828125, -0.0770263671875, 1.8203125, -0.35107421875, 0.470458984375, 0.09063720703125, -0.041351318359375, -0.08123779296875, -0.666015625, -0.134765625, -0.1767578125, 1.123046875, -0.350830078125, -0.248779296875, 1.6123046875, -0.33447265625, 0.362060546875, 0.5458984375, 1.943359375, 0.054840087890625, -0.486572265625, -1.02734375, -0.046600341796875, -0.1260986328125, 1.46875, -1.1953125, 0.388671875, 1.7275390625, 1.28515625, -0.591796875, 0.86767578125, 0.217041015625, -0.2476806640625, 1.037109375, -0.41259765625, 0.406005859375, -0.401123046875, -0.0760498046875, -0.35791015625, 0.86962890625, -0.1563720703125, 0.86279296875, -0.81396484375, 1.310546875, -0.7197265625, 1.2626953125, 1.142578125, 1.212890625, -0.1285400390625, -0.22509765625, -1.701171875, 0.448974609375, -1.078125, -0.68701171875, 1.4404296875, -0.85986328125, 1.1025390625, -1.5361328125, -0.383056640625, -2.09375, -0.341064453125, -0.65087890625, 0.0626220703125, 0.05645751953125, 0.6826171875, 2.337890625, -0.383544921875, 0.51953125, -0.64013671875, 0.97998046875, -0.75341796875, -0.140380859375, -0.50146484375, 0.79296875, -0.417236328125, -1.3681640625, 1.1005859375, -1.423828125, 0.015289306640625, 0.7919921875, -0.039794921875, 0.2003173828125, -0.529296875, 1.3740234375, 0.89794921875, -2.009765625, 0.03265380859375, -1.1474609375, -0.45166015625, -0.11688232421875, -1.033203125, -0.003444671630859375, -0.64306640625, -2.21875, 0.62744140625, -0.03546142578125, 2.66796875, -0.806640625, 0.4013671875, 0.35302734375, -0.167724609375, -1.298828125, 1.1025390625, -0.64013671875, 1.8134765625, 1.6923828125, -1.236328125, 0.00699615478515625, -1.1845703125, 1.0302734375, -0.0718994140625, 0.61279296875, 0.89990234375, -0.059722900390625, -0.1865234375, 0.375, 0.6552734375, 0.92724609375, -2.015625, 1.00390625, 0.318359375, 1.3544921875, 0.01230621337890625, 0.0782470703125, -1.478515625, 0.68017578125, -0.70849609375, 0.580078125, 0.517578125, -0.53125, 0.1341552734375, 1.1171875, 1.2978515625, -0.4990234375, 1.2861328125, 0.51806640625, -0.2166748046875, -0.2021484375, 0.141845703125, 0.7919921875, 0.08673095703125, -2.001953125, -1.7265625, -0.07861328125, 0.97216796875, -0.493408203125, -1.630859375, -0.1885986328125, 0.7783203125, -0.56396484375, 0.6640625, 0.90087890625, 0.1976318359375, 0.5693359375, -0.69677734375, 0.05804443359375, -1.1220703125, -0.28271484375, 0.07781982421875, -0.7763671875, -1.4111328125, -0.1405029296875, 2.01171875, 1.5791015625, -0.4287109375, -1.423828125, -0.2005615234375, -1.447265625, -0.0706787109375, -0.226806640625, 0.20654296875, -0.775390625, -1.1318359375, 0.129150390625, 0.927734375, 1.228515625, 0.9052734375, 0.875, 0.673828125, 0.79736328125, 0.306396484375, 0.1671142578125, 0.58740234375, -0.81689453125, 1.8720703125, -0.8388671875, 0.040069580078125, 0.352783203125, -0.315185546875, 1.2666015625, -1.623046875, -0.036956787109375, -0.67333984375, -0.58349609375, 0.415283203125, 1.791015625, -0.88134765625, -1.4189453125, 0.806640625, -0.3515625, 0.564453125, 0.9521484375, 0.75830078125, -0.01171875, 0.34130859375, -1.2861328125, 1.470703125, 0.60595703125, -0.383544921875, 0.14013671875, 0.94140625, -0.15966796875, 2.19921875, -0.426513671875, 0.70361328125, -0.2061767578125, -1.3154296875, -0.7255859375, 0.270263671875, -0.12030029296875, 1.5390625, 1.2216796875, -0.50244140625, 1.369140625, 1.7421875, -0.56884765625, -1.27734375, 0.7451171875, -1.330078125, 0.72607421875, -0.80615234375, -0.0016803741455078125, -0.24658203125, 1.123046875, 0.8466796875, -1.6767578125, 0.13134765625, -0.90478515625, 0.6533203125, -0.418212890625, -0.0171661376953125, 1.3994140625, -1.208984375, -0.97119140625, -0.681640625, 1.4580078125, -0.0209503173828125, 1.4248046875, 1.03125, 3.1796875, 0.5634765625, 0.66650390625, -0.7099609375, -0.389892578125, 0.41845703125, 0.64697265625, -0.059844970703125, -0.66357421875, 1.5595703125, -1.4658203125, 1.1669921875, -1.4150390625, -0.3388671875, -0.380859375, -0.76123046875, 1.1376953125, 1.2763671875, -1.203125, -0.367431640625, -0.8857421875, -0.11492919921875, -0.021636962890625, 0.07122802734375, -1.1767578125, -0.4375, 1.650390625, -2.115234375, 0.4716796875, 1.6669921875, 0.2357177734375, 0.97412109375, -0.393310546875, -0.7685546875, 0.284423828125, 0.374267578125, -0.66943359375, -0.65380859375, -2.6015625, -0.33837890625, -0.2467041015625, 0.3134765625, -0.1522216796875, -0.9482421875, 0.07623291015625, 0.37841796875, 0.712890625, 0.92529296875, 0.5693359375, -0.9228515625, -1.400390625, 0.298095703125, -0.892578125, 1.341796875, -0.022857666015625, 1.328125, 0.27490234375, -0.447998046875, 0.0826416015625, -1.1083984375, 1.2314453125, 2.451171875, 0.7021484375, -0.8046875, 0.24951171875, -1.3291015625, 0.78369140625, 1.6416015625, 1.5693359375, -1.3037109375, 0.67236328125, 0.01934814453125, -0.297607421875, -0.12054443359375, -0.6904296875, -1.0361328125, 0.62548828125, -1.1689453125, -0.7802734375, 1.015625, -0.78515625, -1.4150390625, -0.580078125, 2.140625, -2.53515625, 0.3818359375, 0.6728515625, -0.45703125, -0.216064453125, -0.417236328125, -1.2841796875, -1.205078125, -0.87109375, 0.9169921875, 0.4638671875, 1.638671875, 0.447021484375, -1.2802734375, 2.013671875, -1.0576171875, 0.69091796875, 1.2041015625, -0.595703125, 0.6875, 1.0439453125, -1.3798828125, 0.2396240234375, 0.319091796875, -0.7509765625, 0.438720703125, -0.34326171875, -0.44775390625, 0.107666015625, 0.79541015625, -0.90625, 0.292236328125, 0.88037109375, 0.0797119140625, 0.71728515625, -0.92529296875, 0.123779296875, -1.4189453125, 0.7529296875, -0.10992431640625, 2.904296875, -0.181640625, 0.6416015625, 0.96240234375, -0.0158843994140625, -0.99365234375, -0.0079803466796875, -0.352294921875, 0.2012939453125, 1.5390625, -0.9130859375, 0.623046875, 0.671875, 0.66650390625, 0.036376953125, 0.2120361328125, -1.8447265625, 0.37841796875, 1.3662109375, 0.07965087890625, 0.85791015625, -0.96240234375, -0.77197265625, 0.7333984375, 1.3466796875, -1.060546875, -0.09307861328125, 0.75634765625, 0.3369140625, -0.72216796875, 0.77587890625, 0.6826171875, 0.84765625, -2.046875, -0.88330078125, 1.40234375, -0.91845703125, -0.174560546875, 0.1290283203125, 0.75390625, 0.7275390625, -0.341552734375, 0.1763916015625, -0.2015380859375, -0.7333984375, -1.1201171875, -0.051300048828125, -0.1824951171875, -1.9404296875, -0.34765625, -0.43701171875, -0.806640625, 0.468994140625, -1.677734375, -1.201171875, 1.4716796875, -1.6328125, 0.8369140625, -0.640625, -0.94091796875, -0.58984375, 0.0931396484375, 1.1259765625, 1.9677734375, 0.677734375, -0.97509765625, 0.37890625, 1.7783203125, 0.65283203125, 0.186279296875, 0.65380859375, 0.64404296875, -1.80078125, 0.2242431640625, -1.1015625, 0.450439453125, 0.365234375, -0.423095703125, 1.16015625, 0.0594482421875, 1.033203125, -1.73046875, 1.0166015625, 1.0205078125, 0.71142578125, 2.2890625, -0.787109375, -0.477783203125, -0.78125, 1.1162109375, -0.47802734375, 0.9296875, -1.037109375, -0.2266845703125, -0.0887451171875, -0.9833984375, 0.65966796875, 1.8720703125, 0.74267578125, 0.40966796875, 0.697265625, 0.9638671875, 0.767578125, 0.666015625, -0.50244140625, 0.6171875, -1.1689453125, 0.248291015625, -2.765625, 1.8544921875, 0.59912109375, -2.185546875, -0.9873046875, -0.09820556640625, 1.8017578125, 1.2880859375, -0.16259765625, -0.47607421875, -0.92626953125, 0.261962890625, -0.7119140625, -0.7724609375, 0.156494140625, 0.248291015625, 0.5185546875, 0.004329681396484375, 0.40234375, -1.306640625, 0.16748046875, 0.89501953125, -0.28564453125, -1.4287109375, 0.58447265625, -0.1961669921875, 0.196044921875, 0.51416015625, -0.365234375, 0.783203125, 0.5185546875, -0.1531982421875, -0.6220703125, 0.7470703125, 0.6171875, -0.370849609375, -0.07830810546875, -0.2095947265625, -0.5146484375, 1.83984375, 0.072998046875, -1.3330078125, 1.2373046875, 1.2255859375, -0.9462890625, 0.80517578125, -0.587890625, 1.123046875, -0.54638671875, -0.046905517578125, 0.0377197265625, 0.1207275390625, 0.85302734375, -0.88232421875, 0.66162109375, 0.998046875, -1.767578125, 1.060546875, 0.326416015625, -0.9931640625, 1.0478515625, -0.270263671875, 0.82958984375, -0.2373046875, -0.415771484375, -1.033203125, -0.62646484375, 1.0703125, -0.165283203125, -1.3408203125, -1.228515625, -0.37255859375, 0.93408203125, -0.344482421875, -1.236328125, -0.267578125, -2.37890625, 0.544921875, 0.6005859375, -0.1431884765625, 0.136962890625, -0.01094818115234375, -0.254150390625, -2.564453125, -0.19287109375, -1.3359375, 1.28515625, -0.955078125, 0.7783203125, 0.293701171875, -0.73779296875, -1.2861328125, 0.34912109375, 0.339599609375, -0.1087646484375, -0.1199951171875, -0.46923828125, 0.94384765625, -1.173828125, -1.9013671875, -0.96044921875, 0.8623046875, 0.2178955078125, 1.1728515625, 0.491943359375, 1.6552734375, -0.93505859375, 0.22705078125, -0.41845703125, 0.08380126953125, -0.176513671875, 1.65234375, -1.53515625, 0.58056640625, -1.9208984375, 0.6064453125, -0.7568359375, 0.7958984375, 1.0546875, -0.3359375, 1.24609375, -1.109375, 2.373046875, -1.255859375, -0.257080078125, -0.39892578125, -1.0126953125, -0.008880615234375, -0.79541015625, 0.611328125, -0.57861328125, 0.99755859375, 1.2080078125, 0.054901123046875, -2.552734375, 0.491455078125, -0.26025390625, 1.77734375, 0.119384765625, 0.261474609375, 1.3427734375, -0.1783447265625, -0.662109375, 0.054351806640625, 0.52734375, 0.168701171875, -0.06329345703125, -0.1927490234375, 0.00428009033203125, 1.8876953125, 0.7265625, -1.8681640625, 0.54638671875, 0.50244140625, -0.5146484375, -0.646484375, -0.4951171875, 0.58544921875, 0.267333984375, -1.5830078125, -0.167724609375, 2.330078125, 1.248046875, 0.76220703125, 0.29931640625, -0.54150390625, -1.8828125, -0.467041015625, 1.2099609375, 1.2451171875, -0.392333984375, 0.90478515625, 1.755859375, -0.56689453125, 0.681640625, -1.12890625, -1.12890625, 0.15185546875, -1.5751953125, 0.79248046875, -0.09893798828125, -0.8232421875, 2.3671875, 0.384765625, 1.2685546875, -0.6435546875, -0.5849609375, 0.1375732421875, -0.51220703125, -1.58984375, 1.1416015625, 0.51220703125, -0.91064453125, 1.90625, 1.7626953125, 1.6845703125, -1.0693359375, 0.95068359375, -0.2646484375, 0.89892578125, 1.1650390625, 0.55859375, -0.5283203125, -0.68505859375, 1.01171875, 1.556640625, -1.1689453125, -0.331298828125, -0.63671875, 0.314453125, 0.0245208740234375, -1.599609375, 0.194580078125, 0.23974609375, 0.57763671875, 1.05078125, -0.029266357421875, -1.693359375, 1.578125, 0.0648193359375, -0.76611328125, 0.237548828125, -1.3740234375, -0.9677734375, 1.998046875, 0.2406005859375, -0.67822265625, -0.5634765625, 1.3837890625, 0.043975830078125, -0.244384765625, 0.716796875, -0.59423828125, 1.1708984375, -0.69482421875, 0.72802734375, 0.96044921875, 0.51220703125, 0.1973876953125, -0.92626953125, 0.7060546875, -0.287841796875, -0.31787109375, -0.87158203125, 1.4443359375, -0.82958984375, 0.411865234375, -0.293212890625, 0.3173828125, 0.16064453125, -1.3984375, 1.2529296875, 1.6142578125, 1.49609375, 0.385009765625, -1.00390625, -0.81689453125, -0.391845703125, 0.89306640625, 1.6142578125, -0.01470947265625, -3.11328125, 0.52783203125, 1.2451171875, 1.248046875, -1.763671875, 1.373046875, -1.234375, -0.79736328125, 0.23828125, 0.255615234375, 0.7919921875, -0.1788330078125, -1.28515625, 0.55029296875, 0.38818359375, -1.744140625, 0.2685546875, -0.947265625, 0.5927734375, 0.1435546875, 0.462890625, 2.2109375, -0.199462890625, 0.951171875, -0.71484375, 0.8681640625, -1.10546875, -1.09765625, 1.3623046875, 0.2481689453125, 0.92138671875, -0.93896484375, 1.8251953125, -2.654296875, 1.34765625, -0.086669921875, -0.5478515625, -0.79931640625, 1.15234375, -2.337890625, -0.05621337890625, 0.556640625, -1.6826171875, -0.15380859375, -0.50341796875, -0.306396484375, 3.1171875, -1.8154296875, 0.0009288787841796875, 0.321044921875, 0.47119140625, -0.152587890625, 0.043243408203125, -0.38134765625, -0.8388671875, -2.48828125, -1.58984375, 0.8427734375, 1.34375, 0.09576416015625, -0.211181640625, -0.50244140625, -0.52880859375, -1.5146484375, -1.0947265625, -1.2529296875, 0.28369140625, 0.3330078125, -0.53955078125, 2.919921875, 0.429931640625, 0.293212890625, 0.080810546875, -2.177734375, 0.363525390625, 1.654296875, -1.552734375, 1.4326171875, 1.115234375, -0.0150299072265625, 0.52099609375, -0.58349609375, 0.91162109375, -0.92431640625, 1.9560546875, -0.68603515625, -0.52001953125, 0.0186004638671875, 1.1201171875, 0.1827392578125, -0.54443359375, -0.98046875, 0.407470703125, 1.3857421875, 0.24169921875, 0.0628662109375, -1.720703125, -1.5595703125, -0.50439453125, -0.2763671875, 0.232666015625, -0.451416015625, -0.60498046875, -2.435546875, 2.048828125, -0.8251953125, -1.4228515625, 0.4560546875, -0.328369140625, -0.1405029296875, 0.48486328125, 0.82177734375, -1.6328125, -1.00390625, 0.71142578125, -1.5927734375, -1.40234375, 0.319091796875, 0.371826171875, -2.826171875, 0.1737060546875, -1.2236328125, 0.9921875, 0.49365234375, 0.08453369140625, -0.59130859375, 0.52880859375, -0.35498046875, 0.494384765625, -0.407958984375, 0.1927490234375, -0.128173828125, 2.162109375, -1.4013671875, -1.259765625, 0.06939697265625, 1.166015625, -0.13134765625, -0.84326171875, 0.34423828125, -0.787109375, 1.7080078125, 0.1861572265625, 1.7880859375, 1.03515625, -0.399169921875, -0.111572265625, -1.041015625, 1.654296875, -0.892578125, 0.357421875, 1.556640625, 0.53515625, 0.2100830078125, 1.025390625, 1.5830078125, 0.302978515625, 0.52490234375, 0.4091796875, -0.798828125, 0.178955078125, -2.0234375, -2.6640625, -0.7431640625, -0.70458984375, 0.5068359375, -0.71142578125, 1.072265625, 0.1583251953125, -0.79296875, 2.140625, 0.79150390625, 0.56298828125, -0.118408203125, 0.187255859375, 0.8720703125, 0.67626953125, 1.14453125, -0.25830078125, 0.6708984375, 1.7373046875, 0.64111328125, -0.398681640625, -0.0223846435546875, 1.029296875, -0.73779296875, 0.4833984375, 1.8935546875, 0.1746826171875, 0.87646484375, -0.62060546875, -0.48974609375, -0.83349609375, -1.103515625, -0.4873046875, -0.1336669921875, 0.09600830078125, -0.4912109375, 1.1640625, -0.169189453125, 0.041229248046875, 0.1448974609375, 0.385986328125, 2.650390625, -1.8603515625, 1.3701171875, -0.1370849609375, -0.66552734375, 0.11334228515625, -0.58935546875, -0.78857421875, 0.18115234375, 0.01910400390625, -0.46923828125, 1.1865234375, -1.138671875, -0.177978515625, 0.387939453125, -0.2724609375, -0.29296875, 0.426513671875, 0.685546875, -0.54296875, 0.004795074462890625, 0.30615234375, 1.380859375, -2.28125, -1.755859375, 1.4599609375, -0.39697265625, -0.8486328125, 0.24658203125, -0.1063232421875, 0.2169189453125, -0.1817626953125, 0.480224609375, -1.54296875, -0.40234375, 1.2939453125, -0.60205078125, -2.236328125, -2.771484375, 0.92724609375, -1.16015625, 0.168212890625, -1.6044921875, -1.4052734375, 1.146484375, 0.02215576171875, 1.2001953125, 0.6025390625, 0.0090789794921875, -0.416748046875, -1.603515625, -0.5400390625, -1.0986328125, -1.0703125, 0.78369140625, -0.1365966796875, -0.732421875, 0.10125732421875, 0.99560546875, 0.708984375, -0.00015497207641601562, -0.46826171875, 0.462646484375, 0.90625, 0.348876953125, 0.36572265625, 0.89208984375, -1.0419921875, -0.469970703125, 0.42578125, -0.00579071044921875, 0.1512451171875, -0.8271484375, -0.232421875, 1.0947265625, -2.05859375, -0.181884765625, -0.876953125, 0.4755859375, -0.2430419921875, 2.2578125, -0.043487548828125, -1.8974609375, -2.388671875, 0.443359375, 0.67626953125, -1.0693359375, -0.615234375, -1.6240234375, 0.60107421875, -0.43408203125, -0.8046875, -1.2021484375, 0.5498046875, 0.6455078125, 0.45166015625, -0.2457275390625, -0.389892578125, 0.68994140625, -2.125, -1.47265625, 1.2841796875, 0.059234619140625, -1.4892578125, 0.033355712890625, 0.828125, -1.0126953125, -0.401611328125, -0.546875, 0.406494140625, 2.48046875, -1.302734375, -0.275146484375, -0.6982421875, -0.86572265625, -1.494140625, -1.3642578125, -0.6611328125, -0.2047119140625, 1.9228515625, -1.5693359375, 1.76171875, 0.83203125, -0.68017578125, 0.630859375, 0.330322265625, -0.1639404296875, 0.322021484375, -0.47607421875, -1.798828125, -1.001953125, -1.4716796875, -2.1953125, 0.267578125, 2.1328125, 0.367431640625, 0.62255859375, 0.280517578125, -1.0703125, 1.916015625, 0.418212890625, 0.1495361328125, 0.8583984375, 0.1072998046875, 0.1973876953125, -1.1884765625, -0.039093017578125, -0.84228515625, -0.282958984375, -0.01134490966796875, -0.2030029296875, -0.865234375, -1.1025390625, -0.6083984375, 0.1453857421875, -0.5283203125, 0.9794921875, 1.669921875, -1.4443359375, 0.13818359375, -0.341064453125, 0.347412109375, -1.365234375, 2.345703125, -0.15576171875, 0.7021484375, -0.2646484375, -2.78515625, 0.787109375, 0.63623046875, 0.335693359375, -0.34326171875, -0.66455078125, -1.3359375, -1.3583984375, -0.392578125, 0.80859375, -1.5166015625, 1.5341796875, 0.35107421875, 0.970703125, -0.46533203125, 0.70166015625, 0.226318359375, 1.984375, -1.134765625, -0.0927734375, -0.5478515625, -0.29345703125, -1.54296875, -0.1439208984375, 1.47265625, 0.464111328125, 0.755859375, 1.529296875, 0.031829833984375, -2.044921875, 0.033294677734375, 1.75390625, 0.6455078125, 0.51708984375, -1.4365234375, -1.44140625, 0.6904296875, 0.6953125, -1.443359375, 1.7197265625, -1.2607421875, -1.3544921875, -1.1220703125, 0.62646484375, 0.71142578125, 0.304443359375, 1.5791015625, -1.439453125, 1.056640625, -2.53125, -0.7451171875, 2.12890625, 1.009765625, -1.3359375, 0.390625, -0.50732421875, 0.318603515625, 1.7763671875, 0.0256805419921875, -1.927734375, -0.55712890625, 0.004459381103515625, 0.401123046875, 0.052947998046875, 0.7978515625, 1.322265625, 0.234375, -0.340087890625, 0.6669921875, 0.94775390625, 0.393310546875, 0.90380859375, 0.6953125, 1.3818359375, -0.869140625, -0.72509765625, 1.3583984375, 0.81201171875, -1.1884765625, 1.486328125, 0.216796875, -1.0283203125, 0.0017108917236328125, -0.228759765625, 1.470703125, 0.67578125, -0.3681640625, -0.2685546875, 0.95068359375, 0.07159423828125, 1.7158203125, 0.66748046875, 0.026123046875, 1.62109375, 2.4453125, 1.04296875, -1.6689453125, -0.352294921875, 0.1702880859375, 0.285888671875, -0.58935546875, 0.11639404296875, -0.239501953125, -0.34326171875, -0.8359375, -1.0927734375, 0.54541015625, -0.541015625, -0.92431640625, 1.69140625, -0.30029296875, 1.6748046875, -0.185302734375, 1.037109375, -0.96435546875, 0.317138671875, -1.3798828125, 1.662109375, -1.5078125, -0.9501953125, 1.5078125, 0.1802978515625, -1.03515625, 1.2177734375, -1.6416015625, 1.0595703125, -0.591796875, -0.18359375, 0.54833984375, 0.95068359375, 1.40234375, 1.185546875, -0.51416015625, 0.21142578125, 0.433837890625, 1.2578125, -0.64697265625, 0.64404296875, -1.7109375, -1.25, -1.396484375, 1.4912109375, 0.404541015625, -1.8447265625, 2.1640625, -0.51025390625, 1.4921875, -0.71337890625, 0.58056640625, 0.332275390625, -0.03216552734375, -1.60546875, 1.2421875, -0.240966796875, 0.6826171875, -2.318359375, -0.56640625, -0.409423828125, -0.00867462158203125, 0.1468505859375, -1.8837890625, -0.54248046875, -1.58203125, -0.2432861328125, -0.7099609375, -0.5048828125, -0.3984375, -0.048248291015625, 1.8828125, 0.822265625, 0.6435546875, 0.1055908203125, 0.294189453125, 1.333984375, -0.32666015625, -0.3486328125, -1.26171875, -0.3056640625, -0.420166015625, -1.2294921875, 0.01404571533203125, -1.0322265625, 0.152099609375, 0.21435546875, 0.451904296875, 0.335693359375, -0.125244140625, 1.5146484375, -0.75732421875, -0.11126708984375, 0.80908203125, 0.451171875, 0.72412109375, -0.79443359375, 2.240234375, -1.234375, 0.986328125, 1.2373046875, -0.62255859375, -0.8876953125, 1.435546875, -0.921875, -0.61376953125, 1.0771484375, 0.124755859375, 0.81982421875, 0.258056640625, 0.591796875, 0.495849609375, 1.6015625, -0.81005859375, 0.63427734375, -1.7626953125, -0.1563720703125, 1.3779296875, 0.1451416015625, -0.806640625, 0.9013671875, -0.87548828125, 0.425048828125, -0.38037109375, 1.291015625, 0.08697509765625, 1.2255859375, 0.96533203125, -0.640625, -0.25244140625, 1.1572265625, 0.6435546875, -0.4306640625, -0.1856689453125, 0.53125, -0.04278564453125, 0.22998046875, -1.873046875, -0.007266998291015625, 1.8671875, -0.6494140625, 0.1605224609375, -0.383544921875, 0.172607421875, -1.529296875, 1.87109375, -0.1114501953125, 0.12384033203125, 0.281982421875, 1.5458984375, 1.384765625, 0.73779296875, 1.35546875, 0.947265625, 0.494873046875, -1.6845703125, 0.017333984375, -1.0244140625, -0.03240966796875, -0.85546875, -0.2685546875, 0.9306640625, 0.428466796875, -0.0880126953125, -0.22998046875, -0.5869140625, -0.30615234375, -1.5751953125, -1.00390625, -1.3662109375, -0.1182861328125, -0.89892578125, -0.6533203125, 1.18359375, 0.0909423828125, 0.75439453125, -2.498046875, 0.193603515625, -1.0380859375, -0.70703125, 1.4755859375, 0.87353515625, 0.01113128662109375, 0.11004638671875, 0.485595703125, 0.822265625, -0.40087890625, -0.794921875, 0.72802734375, -1.5009765625, 1.32421875, 0.58203125, -0.6923828125, -0.5751953125, -0.166259765625, 0.716796875, 0.73046875, 1.5751953125, 0.70703125, 0.281005859375, -1.4892578125, 1.2470703125, 0.552734375, 0.56689453125, 0.65576171875, 1.962890625, 1.3623046875, 1.064453125, -0.46533203125, 1.3564453125, -0.609375, -0.6064453125, 1.21484375, 0.12646484375, -1.974609375, -1.498046875, -0.46435546875, 0.87646484375, -1.4326171875, -1.2099609375, 1.4365234375, -1.169921875, -0.348388671875, -0.30615234375, 1.3837890625, -0.55908203125, -0.41357421875, -1.1240234375, 0.474609375, 0.23095703125, 0.40771484375, 0.96923828125, 1.5263671875, 0.64111328125, 0.97509765625, 1.8076171875, -0.80126953125, -1.5517578125, 0.3759765625, 0.84521484375, -0.77197265625, 0.6669921875, -0.55615234375, 0.97216796875, 0.7490234375, -1.5234375, -0.428955078125, -0.05462646484375, -0.247802734375, 0.220703125, 0.211669921875, 0.97900390625, 0.12255859375, 0.75537109375, 1.515625, -0.425048828125, -0.75390625, 0.53271484375, 0.121337890625, 1.3955078125, 1.5517578125, 0.57177734375, 0.131591796875, 1.1640625, -0.3681640625, -1.35546875, -1.05078125, -1.7763671875, -0.2861328125, 0.58251953125, -0.7373046875, 0.312744140625, 0.86669921875, -1.7080078125, -0.290283203125, -0.8642578125, -3.119140625, -1.4365234375, -1.1669921875, -0.5927734375, 1.173828125, 0.7109375, -2.763671875, -0.11492919921875, 0.66943359375, -0.60888671875, -1.5283203125, 0.40576171875, 0.236572265625, 0.11688232421875, 1.15234375, 0.4375, -0.055694580078125, -2.125, -0.9404296875, -1.5400390625, 0.9794921875, 0.1376953125, 1.3349609375, 0.95947265625, -1.65625, -0.327880859375, -1.1865234375, -0.0181121826171875, 0.72216796875, 0.3115234375, -0.8369140625, -0.50048828125, -1.2177734375, -0.72021484375, 0.1162109375, 1.0634765625, -0.031829833984375, -1.23828125, -0.578125, -1.1787109375, 0.13916015625, 1.416015625, 0.75634765625, 0.421875, 0.806640625, -0.6533203125, -0.69287109375, -0.16845703125, 0.92236328125, -0.8515625, -0.95849609375, 0.5234375, -0.71533203125, 0.09136962890625, 0.392578125, -0.360595703125, -1.767578125, -1.6064453125, -1.1953125, 0.444091796875, -0.556640625, 0.05462646484375, 1.1376953125, -0.306640625, -1.5576171875, -0.73779296875, -0.57958984375, 4.484375, -1.6064453125, -0.8173828125, 0.7353515625, -0.1575927734375, -0.12200927734375, -0.0009655952453613281, 1.9189453125, -2.390625, 0.52001953125, -0.93212890625, -0.3291015625, 0.383056640625, 0.7236328125, 0.27197265625, -0.47900390625, -0.0088653564453125, 0.61328125, -0.489501953125, 1.587890625, 1.01171875, -1.0419921875, 0.1024169921875, -0.16796875, 0.094482421875, 1.2177734375, 1.177734375, 0.357177734375, -0.69384765625, 0.095458984375, 1.0341796875, 1.216796875, -0.87255859375, -0.6650390625, -1.33984375, 1.1630859375, 0.47412109375, 0.420654296875, 1.396484375, 1.63671875, 1.2294921875, 0.01296234130859375, -0.1490478515625, -0.348876953125, 0.52978515625, -2.1171875, -2.447265625, -0.4931640625, 0.72607421875, -0.48974609375, 1.2607421875, -0.69873046875, 0.86767578125, 0.419921875, 0.213134765625, -0.0892333984375, 0.08447265625, -0.4658203125, 1.6875, -1.857421875, 0.28466796875, -1.0244140625, -0.1295166015625, -0.33251953125, -0.059173583984375, -0.91162109375, -1.380859375, -0.74609375, -0.1583251953125, 0.25537109375, -1.529296875, 1.619140625, 1.0859375, 0.340576171875, -0.74169921875, 0.80126953125, 0.32080078125, -0.1619873046875, 0.32373046875, 0.283203125, 1.4208984375, 0.290771484375, -1.1875, 0.420166015625, 2.4921875, 0.285888671875, 1.1083984375, 0.416259765625, 1.3154296875, 0.54052734375, 0.05426025390625, -0.29052734375, -0.62939453125, 1.6318359375, 0.62646484375, -1.421875, 1.365234375, -0.94482421875, -1.7451171875, -0.466064453125, -0.88232421875, 0.423828125, -1.001953125, 0.5712890625, 1.939453125, 0.189697265625, 1.287109375, 0.40966796875, 0.91552734375, 0.79736328125, 0.3720703125, 0.6162109375, 1.2841796875, 0.237060546875, 0.19140625, 0.2188720703125, -1.3701171875, 0.07586669921875, -0.391845703125, -1.1083984375, 1.1982421875, -1.359375, -0.412109375, -0.397216796875, 1.830078125, -0.124267578125, -1.2001953125, -0.37109375, -0.42822265625, -0.183837890625, 0.213623046875, 2.255859375, 0.2012939453125, 0.445068359375, 1.7138671875, 1.2861328125, -0.2607421875, 1.4521484375, 1.029296875, 0.50927734375, -0.51806640625, -0.513671875, -0.1746826171875, -1.435546875, 0.91943359375, -0.1885986328125, -0.083740234375, -0.44384765625, 1.0986328125, 0.433349609375, -1.6591796875, -0.0243988037109375, 1.0234375, 0.98388671875, 0.51171875, -0.6669921875, -0.09716796875, -0.06488037109375, -0.9501953125, 1.46875, 1.04296875, -0.84375, -0.260498046875, -0.10089111328125, -0.88671875, 0.138671875, -0.40283203125, 0.3037109375, -1.732421875, -1.3857421875, 0.454345703125, -0.383544921875, 0.28564453125, 0.298828125, -0.9736328125, 0.70263671875, -0.425537109375, -0.70361328125, 1.1923828125, 1.2451171875, 1.3291015625, -0.0268096923828125, -0.31884765625, -0.0244293212890625, 0.30517578125, -0.360595703125, -0.3232421875, -0.1348876953125, -1.5224609375, 1.3271484375, -0.69189453125, 1.078125, 0.87109375, 0.43212890625, -2.107421875, 0.68310546875, -1.9072265625, 0.09344482421875, -0.1766357421875, -0.82177734375, 1.1669921875, 0.36865234375, 1.0205078125, -1.4921875, 0.8828125, -0.06060791015625, -0.2298583984375, 0.7568359375, -0.634765625, 0.269775390625, -0.68212890625, -0.5849609375, 1.1552734375, 1.75, 0.1348876953125, 1.05859375, 0.68896484375, 0.90478515625, 1.0927734375, -0.37255859375, -1.134765625, -1.560546875, -2.197265625, -0.90771484375, -0.2509765625, -1.04296875, -0.626953125, 0.255126953125, -0.468505859375, 0.422607421875, -0.939453125, -2.392578125, 0.318603515625, -0.50537109375, 0.165283203125, 1.265625, -0.034759521484375, -0.55859375, -0.0697021484375, -1.314453125, 0.71630859375, -0.73193359375, 0.083251953125, 0.046783447265625, -0.2841796875, 0.109130859375, 0.34130859375, -0.004199981689453125, -1.0185546875, 1.025390625, -0.96337890625, -0.5400390625, 0.64404296875, -0.037017822265625, -0.45556640625, -0.26318359375, -0.7373046875, 0.2430419921875, -0.0821533203125, -0.9345703125, 1.2158203125, 1.57421875, 0.5068359375, -1.82421875, 1.2197265625, -1.01171875, 0.1258544921875, -0.03997802734375, -0.96337890625, -0.9228515625, -1.046875, -1.498046875, 1.3203125, 0.66943359375, -0.1536865234375, -1.3427734375, -0.96875, 0.6435546875, 0.37158203125, -0.513671875, 1.322265625, -0.2100830078125, 2.69921875, 0.34326171875, 1.099609375, -0.77685546875, 2.07421875, -0.74560546875, 1.0771484375, -0.10223388671875, 0.497314453125, -0.81787109375, -0.0594482421875, -0.47802734375, -0.0953369140625, -1.2041015625, -0.70751953125, 0.457275390625, 1.7216796875, -2.427734375, 0.93896484375, 0.2198486328125, -0.783203125, 0.36962890625, -0.269775390625, 0.2037353515625, 0.308837890625, -1.056640625, -0.61669921875, -1.47265625, 0.044281005859375, -0.23046875, -0.34033203125, -0.2178955078125, 0.76171875, 0.347412109375, 1.083984375, -0.61474609375, 2.28125, -0.06524658203125, 0.1866455078125, -1.599609375, 0.0679931640625, 0.1678466796875, -0.71533203125, -0.16455078125, -0.42041015625, 0.50537109375, 0.05255126953125, -2.5546875, -0.9345703125, 0.93359375, -0.35986328125, -1.0068359375, -0.2100830078125, 0.1524658203125, -0.5166015625, 0.77685546875, -0.81689453125, -1.001953125, -1.2001953125, 0.257080078125, 1.4560546875, 0.116455078125, 1.87890625, 1.5302734375, 2.08203125, -0.51611328125, -0.048553466796875, -1.294921875, -0.0010175704956054688, 0.11749267578125, 1.7294921875, -1.4521484375, -0.916015625, 0.78955078125, -1.5244140625, -0.47705078125, 0.259521484375, -1.142578125, -1.625, -0.6953125, 0.313720703125, -0.720703125, -0.46630859375, -0.119384765625, -1.001953125, -0.90966796875, 0.927734375, 0.0643310546875, -0.9892578125, -1.21484375, 0.35693359375, -0.7119140625, 0.054046630859375, -1.5, -0.49951171875, -0.96630859375, -1.05859375, 2.599609375, -0.49951171875, 0.10809326171875, -0.093017578125, -1.576171875, -1.20703125, -0.283935546875, -0.5966796875, 0.46337890625, 0.76318359375, 0.373291015625, 0.87109375, 1.4326171875, 0.481201171875, -0.57373046875, 0.99609375, 0.4150390625, -0.9833984375, 1.7138671875, -1.2109375, -0.35205078125, 1.2841796875, -0.0982666015625, -1.3212890625, -0.7099609375, 1.9111328125, -0.10491943359375, -1.2431640625, -0.52099609375, -1.048828125, -0.400634765625, 1.365234375, -0.24462890625, -0.55859375, 0.625, 0.8740234375, -0.055206298828125, -1.845703125, 0.45556640625, -0.2861328125, -0.91943359375, 0.619140625, -0.11163330078125, 2.271484375, -0.4970703125, -2.33984375, -0.315185546875, -1.005859375, 1.5009765625, 0.5556640625, -0.444580078125, 0.78759765625, -0.0007929801940917969, -0.173095703125, -0.0872802734375, -0.65771484375, -0.2210693359375, 0.87451171875, 0.041961669921875, 0.8642578125, -0.91259765625, 0.37158203125, -0.64697265625, 0.97216796875, 0.01343536376953125, 0.720703125, 0.7763671875, -1.8994140625, -0.74169921875, -0.9033203125, 1.1103515625, 1.5390625, -0.428466796875, -0.42626953125, 0.92529296875, 0.360595703125, 0.480224609375, -0.81396484375, -0.2744140625, 0.5703125, -0.4677734375, 0.28564453125, 0.1456298828125, 1.9970703125, -1.185546875, 0.42529296875, -0.458740234375, 0.9599609375, 0.11767578125, -0.478271484375, 1.193359375, -0.63720703125, -0.73828125, 0.474365234375, 0.01507568359375, -0.8857421875, -0.340087890625, -0.4482421875, 1.2255859375, -0.736328125, -0.354736328125, -1.1923828125, -0.17529296875, 0.5888671875, -0.841796875, -2.72265625, 1.0927734375, -1.10546875, -0.8720703125, -1.361328125, 0.54443359375, -1.1220703125, -0.07763671875, -1.333984375, -0.10894775390625, 0.1640625, -2.19921875, 0.96728515625, 1.7529296875, -0.8564453125, -0.79443359375, 0.21630859375, 1.5556640625, -0.810546875, -1.7734375, -0.347412109375, 0.5849609375, -1.2119140625, 0.07598876953125, -0.78076171875, 3.30859375, 1.3115234375, 2.064453125, 1.5625, -0.6943359375, -0.55322265625, -0.05279541015625, -1.4638671875, -0.85205078125, -0.105224609375, 0.4560546875, 1.4501953125, 0.52197265625, 0.390869140625, 0.78662109375, -1.2763671875, 1.0693359375, 0.58154296875, -2.083984375, -1.8125, 1.0849609375, -0.337158203125, -1.2919921875, -1.3076171875, -0.8486328125, -1.8701171875, 0.94189453125, -1.3359375, -1.2275390625, -0.284912109375, 1.865234375, 0.2030029296875, -0.387939453125, -0.0245819091796875, 1.0009765625, -0.1671142578125, -1.34765625, -0.08489990234375, 0.377197265625, -0.1094970703125, -0.86962890625, 1.2080078125, -1.0830078125, 1.830078125, 2.0078125, -0.50732421875, 0.537109375, 0.262939453125, 0.513671875, -0.07916259765625, -0.591796875, -2.072265625, -0.9814453125, -0.264404296875, -0.51708984375, -0.1512451171875, 0.452392578125, -1.6650390625, 0.93701171875, -0.21923828125, 2.447265625, -1.4189453125, -0.1551513671875, -0.76513671875, 0.77734375, -0.61572265625, -1.8095703125, 1.6171875, 0.64697265625, 1.041015625, 0.60009765625, -0.033599853515625, -0.293701171875, 0.377197265625, 0.603515625, -0.7109375, 0.1473388671875, -0.127685546875, 0.66015625, 0.42431640625, 0.8662109375, -1.0830078125, -0.273193359375, 0.7138671875, -0.53271484375, 0.8857421875, -0.1654052734375, 2.03515625, 0.57470703125, -0.59033203125, 0.1663818359375, 0.9853515625, 1.9794921875, -0.283447265625, 0.339599609375, -0.56787109375, -0.7783203125, 0.049163818359375, 1.7568359375, 0.93896484375, 1.06640625, 0.39453125, 0.456298828125, -0.66748046875, -0.146728515625, -0.54541015625, 0.54248046875, 0.98974609375, 0.22705078125, -1.9443359375, -1.0869140625, 1.408203125, -0.0662841796875, -1.2783203125, 0.7685546875, 0.269287109375, 0.943359375, 0.9375, -0.8857421875, 0.65185546875, -1.71875, -0.87255859375, 1.1923828125, 0.45849609375, -0.337646484375, 1.2822265625, -1.1357421875, 0.7890625, 0.392333984375, -0.028839111328125, 0.58203125, 1.294921875, -0.151611328125, -1.2744140625, -0.9609375, -0.0114898681640625, -0.68212890625, 0.1507568359375, -0.529296875, -1.0361328125, -0.7890625, -0.061248779296875, -0.86328125, -0.1588134765625, 0.4443359375, 0.1737060546875, -1.287109375, -0.96728515625, -0.1317138671875, 1.3076171875, 1.6318359375, 0.9189453125, 0.73486328125, 2.373046875, -0.215087890625, -1.5517578125, 0.59912109375, -1.328125, -0.62939453125, -0.70751953125, 1.3251953125, -1.5009765625, 0.54296875, -0.64794921875, 0.2430419921875, 0.93896484375, 0.0936279296875, 0.1519775390625, -0.492919921875, -0.1143798828125, 0.1793212890625, -0.61279296875, 0.011688232421875, -0.0169219970703125, 0.0751953125, -1.0859375, -1.34765625, -1.04296875, -0.4287109375, 1.09765625, -0.67626953125, 0.09405517578125, -0.767578125, -0.48876953125, 0.459716796875, 0.89599609375, 0.044464111328125, -0.1463623046875, -0.1873779296875, -0.50439453125, -1.4296875, 1.021484375, 2.109375, -0.20751953125, -0.53955078125, -0.77587890625, -1.8154296875, 0.60107421875, -0.2164306640625, 0.56005859375, -0.456787109375, 0.087158203125, -0.2470703125, 0.5146484375, -1.1318359375, -1.505859375, -0.9755859375, -1.7880859375, -0.412109375, 0.6396484375, -1.01953125, -0.459716796875, 2.197265625, -1.1298828125, -1.8984375, -2.494140625, -1.3603515625, 1.4462890625, 0.79296875, -0.740234375, -0.65185546875, -1.744140625, 0.837890625, -0.97216796875, -0.71826171875, 0.2218017578125, 0.45751953125, -0.255126953125, -2.107421875, 1.0986328125, -0.75, -1.171875, 0.2218017578125, 0.267822265625, -0.23193359375, -1.7255859375, 0.18994140625, -1.4931640625, -0.921875, -0.740234375, -0.9326171875, 0.55908203125, 0.114990234375, -0.490966796875, 0.62158203125, 0.94482421875, 0.0960693359375, 0.57763671875, 0.341552734375, -0.033477783203125, 0.53173828125, -0.374755859375, -1.990234375, -0.52880859375, -2.060546875, -2.8671875, -2.0703125, 0.25830078125, 0.78125, -0.312744140625, 1.55078125, -0.65625, 1.166015625, 0.98388671875, -0.1419677734375, -0.79296875, 1.1025390625, -0.7822265625, -0.58349609375, 1.00390625, 1.046875, 1.0966796875, 1.2294921875, 0.433349609375, 0.057098388671875, -1.4814453125, 1.046875, 1.4716796875, -1.154296875, 0.34326171875, 0.705078125, 1.3212890625, 0.8271484375, 0.251220703125, -0.56494140625, -1.79296875, -0.65625, 1.3408203125, 0.26708984375, -0.84912109375, -1.8046875, -0.11761474609375, -0.30712890625, 0.28759765625, -0.91357421875, 1.2333984375, -1.208984375, -0.1419677734375, -1.1494140625, 1.0361328125, 2.03515625, -1.0810546875, 0.067138671875, -0.60888671875, -2.1015625, -0.3486328125, 0.19482421875, 0.7060546875, 0.8955078125, 0.2822265625, 0.9111328125, -1.0595703125, 0.166259765625, 0.058502197265625, -0.53515625, 0.7978515625, 1.1494140625, -2.291015625, 0.9560546875, 0.31689453125, 0.26416015625, 1.5205078125, -1.2626953125, 0.499267578125, 1.58984375, 0.24951171875, -0.10540771484375, 0.2266845703125, 0.0350341796875, 1.1484375, -1.4306640625, -0.189697265625, 3.3984375, -0.87255859375, -0.2578125, 0.453369140625, -0.194091796875, -0.1590576171875, -0.35791015625, 1.03125, -1.154296875, -0.444091796875, -0.5439453125, 0.184814453125, 1.42578125, -0.818359375, -0.462890625, -0.25146484375, 0.434326171875, -1.18359375, 2.34765625, 0.325927734375, 1.0185546875, -0.428955078125, 0.25341796875, -0.1005859375, -0.2071533203125, -0.105712890625, 0.6865234375, -0.9462890625, 0.430908203125, 0.380126953125, 0.77001953125, 0.8232421875, 0.0523681640625, -0.56298828125, 2.1484375, 0.650390625, 0.3642578125, -0.98046875, -0.68701171875, -0.51513671875, -0.33056640625, -0.93310546875, 0.4892578125, 0.3525390625, 0.69189453125, -1.8076171875, -0.206787109375, -1.12890625, -1.3193359375, -0.61865234375, -0.50927734375, 1.61328125, -0.9599609375, 0.587890625, 0.406982421875, 0.141357421875, -0.4775390625, 0.7763671875, -0.203125, 0.6181640625, -0.5048828125, 0.319091796875, -1.0849609375, 1.8076171875, -0.388427734375, -0.640625, -0.84130859375, -0.8857421875, 0.201171875, -1.0966796875, -0.2274169921875, 1.4248046875, 0.52294921875, -0.45849609375, 0.9423828125, 1.1279296875, 1.9072265625, -0.54296875, -0.2236328125, 0.68310546875, 0.3505859375, -0.1337890625, -0.10986328125, 0.2257080078125, -0.44287109375, 0.1268310546875, -1.4404296875, 0.5771484375, 1.130859375, -1.1435546875, 0.2734375, -0.91650390625, 0.2039794921875, -1.2763671875, 1.9345703125, -3.75, -1.7783203125, -0.9775390625, -0.94873046875, -2.404296875, 0.2073974609375, 2.208984375, -1.251953125, 2.544921875, 0.96875, 0.9375, 0.5439453125, 0.496337890625, -0.52783203125, -0.2008056640625, -0.127685546875, 0.52978515625, 0.135986328125, 0.68212890625, 0.5517578125, 1.5859375, 0.82373046875, -0.439697265625, 0.84033203125, -1.306640625, 0.51025390625, -2.47265625, 1.4443359375, 0.806640625, 0.658203125, 0.1927490234375, 0.25390625, -0.385009765625, 0.271484375, -1.2490234375, 1.0078125, -0.744140625, -1.0810546875, -0.1898193359375, 0.66650390625, -0.90966796875, -1.0771484375, -0.1922607421875, -0.0462646484375, -0.168701171875, 0.29248046875, 0.00042128562927246094, -0.86865234375, 1.1982421875, -0.486083984375, 0.77734375, 0.56298828125, -1.814453125, 0.8525390625, -0.9736328125, 0.8173828125, -0.465087890625, 0.018218994140625, 0.284423828125, 1.798828125, 0.2626953125, -0.175537109375, -0.044830322265625, 0.1572265625, 0.469970703125, 0.1893310546875, 1.123046875, -0.326904296875, -0.6318359375, 1.19921875, 0.51171875, -0.7421875, -0.71728515625, 0.623046875, 0.6669921875, -1.654296875, 1.1748046875, 1.71875, -0.41650390625, -2.037109375, -2.583984375, -0.25732421875, -1.029296875, -0.0285797119140625, 0.21826171875, -1.486328125, -0.461669921875, 1.4658203125, -0.5595703125, -1.5693359375, 0.0823974609375, 0.2958984375, -0.61083984375, 0.85791015625, 0.2293701171875, -0.095458984375, 0.51318359375, 1.4970703125, -1.083984375, 0.380859375, 2.087890625, -0.10064697265625, 2.302734375, 0.3564453125, -1.1552734375, -2.435546875, 1.0166015625, -0.326904296875, -0.6884765625, -0.54833984375, 0.5712890625, 3.111328125, 0.429443359375, 0.52490234375, 0.25537109375, 0.278564453125, -0.329345703125, 0.75537109375, 0.33544921875, -1.353515625, -0.3564453125, 1.1416015625, -2.91015625, -0.31884765625, -1.7607421875, 0.159423828125, -0.0697021484375, -1.0654296875, -1.1142578125, -1.166015625, 1.8388671875, -0.004589080810546875, -0.2919921875, 0.69873046875, -1.41015625, 0.66357421875, -0.026763916015625, -0.54443359375, -1.861328125, 0.86474609375, -0.53955078125, -1.083984375, -1.041015625, 0.2049560546875, -1.0703125, 0.328857421875, 1.7744140625, -0.420166015625, 0.062286376953125, 0.08123779296875, 2.51171875, 0.00557708740234375, -0.95361328125, -0.72314453125, -1.365234375, 0.3857421875, -0.78662109375, -1.24609375, -1.1044921875, 0.1441650390625, 1.333984375, -0.42822265625, -0.1939697265625, 0.353271484375, 0.56640625, -2.123046875, -1.0576171875, -1.0390625, -1.681640625, -0.3837890625, 1.03515625, -1.603515625, 0.285888671875, 0.32177734375, -0.56103515625, 0.78076171875, -0.51806640625, -1.1591796875, -1.244140625, -1.40625, 0.0711669921875, -0.399658203125, -0.70849609375, -0.461181640625, -0.293701171875, -1.4072265625, 0.0826416015625, -0.064453125, 0.349365234375, 0.208251953125, -0.048858642578125, 0.1414794921875, -1.13671875, 1.7373046875, -0.34375, 0.496826171875, 0.751953125, 0.144775390625, -0.454833984375, -0.30224609375, 0.239990234375, 0.1619873046875, -0.39111328125, 1.9599609375, -2.27734375, 0.234130859375, -0.333984375, 1.4462890625, -3.07421875, 0.87939453125, 0.98486328125, 0.01541900634765625, 0.71533203125, -0.1876220703125, -0.0073089599609375, -0.5146484375, 0.53369140625, -0.77783203125, 0.6953125, 0.205322265625, -0.86865234375, 0.300537109375, -2.193359375, -0.11260986328125, -0.369384765625, -0.317138671875, -0.4384765625, 0.51904296875, -1.0439453125, 2.919921875, 0.1048583984375, -1.001953125, -2.20703125, -0.1441650390625, -1.05078125, 0.102783203125, -0.1220703125, 0.201416015625, 1.1953125, 0.490234375, 1.708984375, -3.244140625, 0.299072265625, 0.69580078125, -1.7724609375, -0.2091064453125, -0.368896484375, -0.9169921875, 1.5322265625, -1.091796875, -0.79443359375, 1.060546875, -2.388671875, -1.7978515625, -1.0068359375, 0.568359375, -1.1806640625, 1.48046875, -0.59375, -0.342529296875, 0.1875, -0.9970703125, -0.79541015625, -1.6591796875, -0.56787109375, 0.04693603515625, -0.003246307373046875, -0.09124755859375, 1.5751953125, -0.58251953125, 0.149169921875, 0.1417236328125, -0.65576171875, 0.64892578125, -0.05792236328125, -0.70166015625, -1.333984375, -0.463134765625, 1.427734375, 0.1597900390625, -1.208984375, 0.55615234375, 1.0751953125, -1.7861328125, 0.76904296875, 0.40283203125, 0.0098724365234375, -1.6328125, 0.1148681640625, -0.88134765625, -0.126708984375, 1.96875, -1.423828125, 0.52099609375, -3.220703125, 0.1065673828125, 0.72314453125, 1.6513671875, -0.430419921875, 0.2222900390625, 0.6845703125, -0.0178680419921875, -0.2308349609375, 0.88232421875, 0.787109375, -0.7734375, 0.505859375, 1.4580078125, 1.8515625, 0.01447296142578125, 0.8876953125, -0.6455078125, -0.490966796875, -0.39306640625, -1.1201171875, 0.401611328125, 0.603515625, -0.1546630859375, 0.2333984375, 0.2200927734375, -1.05859375, 0.5615234375, -0.034881591796875, -0.60009765625, -0.1336669921875, -0.93408203125, -0.57421875, -0.4326171875, -1.302734375, -0.450439453125, 1.53125, -0.452392578125, -0.312255859375, 0.8974609375, -0.5546875, -0.281005859375, 0.25732421875, -0.383544921875, -0.417236328125, 0.11053466796875, -2.32421875, 1.248046875, 0.0391845703125, 0.783203125, -0.305908203125, -0.02490234375, 0.0107421875, -0.10272216796875, -0.05767822265625, 0.1796875, -0.036163330078125, 0.37255859375, -0.3525390625, -0.701171875, 0.89013671875, 1.1806640625, -0.043853759765625, -1.8115234375, -1.109375, 0.98974609375, -0.03369140625, 0.411865234375, 0.3720703125, -0.002349853515625, -0.55859375, 0.050201416015625, 0.183837890625, 0.1304931640625, 0.75146484375, -0.0968017578125, 0.68408203125, -0.327392578125, 0.333984375, 0.1533203125, 0.421142578125, 1.1533203125, 0.86376953125, 0.80517578125, -1.26953125, 0.314697265625, -0.09771728515625, 0.0166015625, 1.0458984375, 0.1566162109375, -2.361328125, 1.00390625, 0.57373046875, -0.324462890625, 0.9287109375, -0.229736328125, -0.611328125, -0.363525390625, 0.313720703125, -0.39599609375, 1.3330078125, 0.7705078125, 1.2021484375, -1.494140625, -1.32421875, 0.275390625, -0.67431640625, 0.1707763671875, 0.191650390625, -0.396484375, 0.336181640625, -0.196533203125, -0.237060546875, 1.0947265625, -0.74169921875, -0.71142578125, -0.004650115966796875, -0.2144775390625, 1.4052734375, 0.955078125, 0.3916015625, -0.373046875, -1.7578125, 1.42578125, -0.1622314453125, 0.3544921875, 3.2890625, 0.9375, 1.5068359375, 1.1962890625, 1.076171875, 0.62353515625, 1.208984375, -1.005859375, 0.529296875, 0.65234375, -0.134765625, -1.29296875, 0.205078125, 0.264892578125, 0.79833984375, -1.30078125, 0.2064208984375, -0.07342529296875, -0.1341552734375, 0.7080078125, 1.8681640625, 0.53125, 0.41015625, 0.0345458984375, -0.31103515625, -1.3564453125, 0.67529296875, -0.767578125, 0.1878662109375, 0.96044921875, -0.66943359375, -0.140380859375, -0.4375, -0.83251953125, 1.185546875, 0.529296875, -0.63525390625, 0.97265625, -0.61962890625, -0.463623046875, -1.4482421875, -0.08636474609375, -0.71142578125, -0.84814453125, -1.5751953125, -0.1085205078125, 0.324951171875, 1.2626953125, -1.1806640625, -0.80859375, -0.5751953125, -1.1845703125, -0.64697265625, -0.11346435546875, 0.44970703125, -0.8154296875, -1.154296875, -0.18310546875, -0.1356201171875, -0.68310546875, 1.2685546875, -1.66015625, 0.7109375, -2.3203125, -0.10150146484375, -1.533203125, 0.00981903076171875, -1.3515625, 0.6630859375, -0.0980224609375, -1.0302734375, -1.79296875, 0.1114501953125, -1.0166015625, 0.9501953125, -0.88671875, -0.8466796875, -0.75341796875, 1.498046875, -0.72998046875, 1.7470703125, 1.361328125, -0.89306640625, -0.9677734375, -1.525390625, 1.2900390625, 1.728515625, 1.7705078125, -0.74609375, -0.96484375, -0.240966796875, 0.1253662109375, 0.283203125, 0.009307861328125, -0.12237548828125, -1.7841796875, -0.2578125, -0.78515625, 0.2303466796875, 0.0880126953125, 1.4501953125, -0.247314453125, -0.1904296875, -2.06640625, 0.52294921875, 0.65234375, 0.802734375, -0.431640625, -0.039642333984375, -1.369140625, -0.2430419921875, 0.43603515625, 0.326416015625, -1.0830078125, 1.298828125, 0.56591796875, -0.15673828125, 0.1942138671875, 0.6181640625, -0.984375, 0.1114501953125, -0.455078125, 0.693359375, -1.3662109375, 0.62890625, -1.7529296875, 0.70361328125, -1.7734375, -0.203125, -0.8271484375, 0.6767578125, 0.06573486328125, 0.5830078125, 0.04144287109375, 0.8984375, -0.344970703125, 1.25390625, -0.04486083984375, 0.055450439453125, -0.07867431640625, -0.69287109375, -1.6748046875, -1.1396484375, 0.521484375, 0.13037109375, 0.329345703125, 0.459716796875, 0.34375, -0.43359375, -1.212890625, -0.240234375, 0.364990234375, 1.2265625, 1.0048828125, 0.97265625, 0.33740234375, -0.283203125, 1.873046875, 0.79638671875, -1.458984375, -0.9501953125, 0.30908203125, 0.775390625, -0.986328125, 0.1287841796875, 0.278564453125, 0.2254638671875, 0.341552734375, 0.8544921875, 0.343994140625, 2.083984375, -0.373046875, -0.48486328125, -1.0439453125, -1.5263671875, 1.1220703125, 1.212890625, -0.355224609375, 0.3388671875, -0.055084228515625, 0.1488037109375, -0.0411376953125, 0.37841796875, -0.123779296875, -1.1572265625, 0.254638671875, 0.21142578125, -1.3203125, 0.31298828125, 0.5849609375, -0.37548828125, 0.79833984375, -0.0858154296875, -0.880859375, 1.55859375, -1.5478515625, 0.07452392578125, 0.01129150390625, 0.364501953125, 1.4248046875, 0.24365234375, -0.31201171875, 0.82275390625, -0.5478515625, 0.48876953125, 1.587890625, -2.544921875, -1.1025390625, -0.036376953125, -0.41064453125, 0.81640625, -0.33056640625, -0.057525634765625, 0.89111328125, -0.2216796875, 0.261474609375, 1.9384765625, -0.28125, 1.05078125, 0.78125, -0.2139892578125, 0.06341552734375, -0.76953125, -0.382568359375, 0.8818359375, -1.1455078125, -0.00911712646484375, 0.2529296875, 1.5888671875, -0.50048828125, -0.1849365234375, -1.578125, 0.572265625, -0.338134765625, 0.90869140625, 0.80322265625, -1.2099609375, -0.0975341796875, 0.537109375, 1.1181640625, -1.1728515625, -0.295654296875, -0.364501953125, -1.015625, 0.6875, 1.896484375, -0.040679931640625, -0.57763671875, -0.75634765625, -0.15869140625, -0.269287109375, -0.80517578125, 1.1806640625, 0.390869140625, 1.439453125, 0.3046875, -0.79833984375, -1.375, -0.93310546875, 0.0869140625, 0.72607421875, -0.6279296875, 1.1005859375, -0.61767578125, -0.02618408203125, -0.35986328125, 0.62158203125, 0.5224609375, 1.556640625, -1.09765625, -1.052734375, -1.1201171875, 1.01953125, 0.0010633468627929688, -0.85400390625, -1.9169921875, 0.330810546875, 1.30078125, -1.1845703125, -1.1982421875, 1.0244140625, -0.27294921875, -0.495849609375, -1.130859375, 0.50048828125, 0.560546875, -0.55078125, -0.52099609375, 0.260498046875, 0.9619140625, 1.498046875, -0.073974609375, -0.328369140625, -0.76416015625, -0.11663818359375, 0.1083984375, 0.476806640625, -1.3056640625, 0.849609375, 0.07025146484375, -1.1328125, 0.66455078125, -0.650390625, -0.05462646484375, -1.0517578125, -0.54150390625, 0.281494140625, -0.2164306640625, -1.1884765625, -0.97509765625, -0.140869140625, -2.69921875, -1.171875, -0.78662109375, -1.4521484375, -1.6845703125, 0.260986328125, 0.37548828125, -0.68994140625, 0.28369140625, 0.33349609375, 0.288818359375, 0.88623046875, -0.3330078125, 0.81591796875, 0.133544921875, 0.1827392578125, -0.066162109375, -1.685546875, -0.489990234375, -1.5439453125, -1.33203125, -0.7802734375, -0.93310546875, 0.52587890625, 0.1826171875, 1.3857421875, 0.90478515625, -0.62548828125, -0.56689453125, 1.58203125, -0.82421875, 0.80126953125, -0.79296875, 1.9404296875, 1.24609375, -2.625, 2.275390625, 0.47314453125, 1.0927734375, 1.443359375, 1.3916015625, 0.02740478515625, 0.2109375, -1.0341796875, -0.22998046875, -1.0390625, -0.61328125, -0.250244140625, 0.9677734375, 1.291015625, 0.779296875, -0.763671875, 1.3115234375, 0.408935546875, -0.53955078125, -1.0166015625, 0.0498046875, 1.423828125, 0.057647705078125, -1.216796875, 0.71630859375, 0.473388671875, 1.9609375, 0.347900390625, 0.150146484375, -0.363037109375, -0.85791015625, 1.0546875, 0.8876953125, 0.2215576171875, 0.153564453125, -0.0201416015625, -0.08026123046875, 0.4970703125, -1.1591796875, 1.2958984375, -0.299072265625, 0.95654296875, -0.30859375, -1.1748046875, 1.3349609375, 2.064453125, -1.01171875, 0.101806640625, -0.83203125, -0.5546875, -0.93701171875, 0.07464599609375, -0.054351806640625, 0.8515625, -0.7958984375, -1.1875, 0.1337890625, 0.005374908447265625, 1.591796875, 0.0885009765625, 1.15625, 0.208740234375, -1.8017578125, -0.1517333984375, 0.135498046875, 0.32275390625, -0.13916015625, 0.1845703125, -0.68408203125, 0.0294647216796875, 0.16064453125, 0.767578125, -0.1529541015625, -0.267822265625, -0.185791015625, 0.88818359375, 0.54150390625, 0.322265625, -0.0521240234375, -0.234130859375, -1.0244140625, -0.298583984375, 0.87890625, -0.0455322265625, 0.53369140625, -0.3896484375, 0.19873046875, -1.4501953125, 2.673828125, 0.52587890625, -0.89404296875, -1.103515625, 1.03125, 0.78857421875, 1.4462890625, 0.1392822265625, 2.751953125, -2.53125, -0.69970703125, 1.486328125, -0.0291900634765625, 1.2314453125, 0.837890625, 1.041015625, 0.36279296875, -1.38671875, -0.95263671875, 0.485595703125, -0.56201171875, 1.8916015625, -0.228515625, -0.176025390625, -0.37060546875, 0.32958984375, -0.2783203125, 0.4443359375, 0.3271484375, -1.19140625, 0.8134765625, -0.447509765625, -0.1131591796875, -0.1031494140625, -1.4033203125, -0.267578125, -0.794921875, -1.6328125, 0.1009521484375, 0.06158447265625, -0.06658935546875, 0.50830078125, 0.1500244140625, 0.34375, 1.435546875, 0.759765625, -0.1749267578125, -1.6728515625, 0.12139892578125, 0.9794921875, -1.029296875, 3.423828125, 0.6513671875, 0.658203125, -1.9990234375, 0.420166015625, -1.4541015625, -0.68115234375, 0.66259765625, 0.13525390625, 0.287353515625, 1.177734375, 1.28125, -0.18994140625, -0.389892578125, 0.162841796875, 0.94677734375, -1.26953125, 1.1640625, -1.064453125, -2.240234375, -0.26025390625, -1.0380859375, -1.2939453125, -0.80078125, -0.28271484375, 0.06597900390625, 1.1396484375, -0.7451171875, -0.422607421875, 0.9794921875, -2.296875, 0.791015625, -1.099609375, 1.60546875, -0.1412353515625, 0.455810546875, 0.61572265625, 0.47509765625, 1.46875, 0.335205078125, -0.78466796875, -1.021484375, 0.2086181640625, -0.161376953125, -1.189453125, -0.97119140625, -1.1318359375, 2.92578125, 0.78125, 1.119140625, -1.0029296875, 0.031158447265625, -1.3486328125, 0.51123046875, -0.9833984375, -1.3759765625, -0.20751953125, -0.8544921875, -0.74755859375, -0.2371826171875, -0.301025390625, -1.8798828125, -0.67578125, 0.0197906494140625, 2.29296875, 0.064453125, 0.6416015625, 1.5771484375, -1.4912109375, 0.57275390625, 1.6142578125, 0.0615234375, -0.12335205078125, -0.97802734375, 0.89013671875, 0.0501708984375, -0.90087890625, -1.6943359375, 0.34375, 0.314697265625, 1.1083984375, 1.296875, -0.5859375, -0.29443359375, -1.2490234375, 0.16748046875, -0.0740966796875, -0.267333984375, 1.1162109375, 0.78759765625, 0.1075439453125, 0.45947265625, -1.5859375, -0.12030029296875, 0.103515625, 0.01131439208984375, 0.4169921875, 0.3232421875, -0.276611328125, -0.798828125, -0.98046875, 0.9990234375, -0.98388671875, -0.72265625, 0.138671875, -0.80810546875, -0.55322265625, -2.119140625, 1.642578125, -1.04296875, -1.6435546875, -0.595703125, -0.211181640625, -0.5126953125, -1.3408203125, -2.875, 1.0673828125, -0.52294921875, 1.8447265625, -1.1689453125, 1.24609375, -2.873046875, -0.10272216796875, 0.95068359375, -1.791015625, -0.494140625, -0.087890625, -1.4765625, 1.265625, 0.93017578125, 1.0126953125, -0.17822265625, 1.3935546875, 0.375, -0.048370361328125, 0.05303955078125, -1.451171875, 0.426513671875, 0.81005859375, -2.498046875, -0.185302734375, 1.3544921875, 1.2900390625, -1.7939453125, -0.399658203125, -0.296630859375, -1.2177734375, 0.140625, -0.40234375, -0.91259765625, 1.6279296875, -1.1943359375, -0.07373046875, -1.041015625, -0.1640625, -1.291015625, -0.284423828125, 0.40380859375, -0.0005326271057128906, 1.0283203125, -0.356689453125, -0.1787109375, -1.185546875, 0.84033203125, 0.110107421875, 0.11956787109375, 0.93896484375, -1.83203125, -1.7705078125, 0.7939453125, -0.30859375, -1.09375, -0.9130859375, -0.177734375, 0.2391357421875, 0.7490234375, -0.2286376953125, 1.3330078125, 0.75390625, 0.24755859375, -0.4521484375, -0.046630859375, 1.87890625, 0.386962890625, 0.484130859375, 1.0029296875, 1.3076171875, -0.63720703125, 2.63671875, -0.45166015625, -0.28466796875, 0.9501953125, -0.80517578125, -1.318359375, -0.66162109375, 0.473876953125, 0.421875, 1.5009765625, -0.1973876953125, -0.5888671875, -0.445556640625, -0.642578125, -0.146484375, 0.448486328125, -0.102783203125, -0.127685546875, -0.05108642578125, 0.65673828125, 0.5341796875, -0.0330810546875, -0.6279296875, -0.1390380859375, 0.72509765625, 0.71044921875, 1.8134765625, 0.59765625, 0.45654296875, 0.06817626953125, -0.7314453125, 1.318359375, -0.76806640625, -0.2176513671875, 0.81689453125, 0.1717529296875, 1.5283203125, -1.4228515625, -2.232421875, 0.033660888671875, -0.287841796875, 2.271484375, -0.1109619140625, -0.152099609375, 0.186279296875, -2.126953125, -0.86767578125, -0.040252685546875, 1.6572265625, -2.234375, 1.7666015625, 0.98388671875, 0.3173828125, -0.336181640625, -0.26904296875, -0.65380859375, -1.392578125, 0.1318359375, -0.20068359375, 0.32080078125, 1.208984375, 0.058135986328125, 0.92822265625, -0.6826171875, -1.3037109375, 1.5126953125, -1.0322265625, -0.86572265625, -1.3427734375, -0.85791015625, 1.8515625, -0.1943359375, 1.8466796875, -0.62841796875, 1.7412109375, -0.6171875, 0.255615234375, -0.40625, 0.228759765625, -1.568359375, 0.0240936279296875, -0.52978515625, -0.1683349609375, 0.33935546875, -0.84912109375, 1.2412109375, -0.11749267578125, -0.63916015625, 0.724609375, 0.97509765625, 0.259033203125, -1.9208984375, 0.1805419921875, -0.039581298828125, -1.814453125, -0.10223388671875, -1.44140625, 2.111328125, -0.90771484375, -2.244140625, 0.419921875, 0.3515625, 0.46142578125, -1.5390625, 0.102783203125, -0.13330078125, 1.767578125, -0.0859375, -0.26806640625, 0.90087890625, -0.97705078125, -1.458984375, -0.50439453125, -0.12841796875, 0.2303466796875, -0.26171875, 1.9609375, 0.98828125, -0.99853515625, -2.173828125, -1.552734375, 0.3515625, -0.73193359375, -0.423583984375, -1.2626953125, 0.52685546875, -1.0732421875, 0.86865234375, 0.375, -0.2210693359375, 0.76416015625, 0.8193359375, -0.5537109375, 0.99365234375, -1.103515625, -1.15234375, -0.888671875, -0.181884765625, -0.221435546875, -3.03515625, -0.149169921875, -0.982421875, 2.0234375, -1.5986328125, 0.285888671875, -0.87451171875, 1.095703125, -0.9775390625, -0.14404296875, -1.3994140625, 1.7685546875, -0.9697265625, 0.74560546875, 0.0216522216796875, -0.7158203125, -0.5888671875, 0.05438232421875, -0.54736328125, -0.5244140625, 2.25, 0.82470703125, -0.166259765625, -0.65234375, 0.40087890625, -0.7138671875, 0.66357421875, -0.95556640625, 0.11322021484375, 0.888671875, 0.533203125, 0.78173828125, -0.1787109375, -1.0986328125, -1.189453125, 0.193603515625, -0.4267578125, 0.293701171875, -0.93505859375, 0.59130859375, -0.55908203125, -0.4169921875, -0.0946044921875, 1.37890625, 0.339111328125, 0.5654296875, 2.48046875, -0.84228515625, -0.9697265625, -1.0302734375, -1.3974609375, 1.498046875, 1.2119140625, -0.1617431640625, -0.473876953125, 2.119140625, -0.039306640625, 0.94482421875, 1.98046875, 0.51611328125, 0.04095458984375, -1.228515625, -0.209716796875, -0.10406494140625, 1.1875, -1.30859375, 1.32421875, -1.8115234375, -0.97216796875, 0.052093505859375, 1.2783203125, 0.435302734375, 1.349609375, 1.373046875, 0.32421875, 0.6865234375, -0.10626220703125, -0.75439453125, -0.61181640625, 0.44677734375, -0.71826171875, 0.4365234375, 0.050384521484375, -0.358154296875, -0.95263671875, -0.02313232421875, 0.361083984375, -0.470458984375, -2.64453125, -1.625, 0.4970703125, 1.357421875, -1.447265625, -0.36767578125, -0.42724609375, 0.492431640625, -0.6171875, 0.286376953125, 0.58349609375, -0.08160400390625, -0.52685546875, -0.1768798828125, 0.87060546875, 0.9921875, -0.94580078125, 0.399169921875, -0.1488037109375, 0.72802734375, -0.8330078125, 0.6171875, 0.285400390625, 0.036773681640625, 1.322265625, 1.74609375, 1.0400390625, 0.139404296875, -0.88134765625, -1.373046875, 2.5859375, 1.7861328125, 0.4287109375, 0.9345703125, 1.931640625, 0.293212890625, 0.66943359375, 0.202392578125, -0.026702880859375, 0.460205078125, 1.15234375, -2.115234375, -0.3603515625, -1.7919921875, -0.91064453125, 0.6318359375, -0.56689453125, 1.2314453125, 0.428955078125, -0.0682373046875, 0.5556640625, 2.00390625, -2.13671875, -1.2978515625, 1.970703125, -0.81201171875, -0.59033203125, 0.182373046875, 0.052154541015625, 1.12890625, 0.469482421875, 0.327880859375, 0.72265625, 0.7744140625, -0.8427734375, -0.432373046875, 0.5146484375, 1.443359375, 0.2333984375, -0.427978515625, 0.376953125, -0.91552734375, -0.1885986328125, 0.9326171875, -0.2362060546875, 0.884765625, 1.3427734375, -1.2431640625, -0.2548828125, 1.0830078125, -0.88671875, 1.6611328125, -0.352294921875, -2.39453125, 0.42138671875, 0.77685546875, -1.4013671875, 0.56787109375, 1.4345703125, -1.8623046875, 0.95361328125, 0.66748046875, -0.8671875, 0.461181640625, -0.1002197265625, 0.4453125, 0.67529296875, 0.279296875, -0.0234222412109375, -0.6865234375, 0.794921875, 1.3623046875, -0.445068359375, -0.337158203125, 0.210205078125, 0.529296875, -0.7705078125, 0.3427734375, -0.35546875, 1.2744140625, 1.4697265625, 1.1484375, -0.0185089111328125, 0.352783203125, 1.7099609375, 0.107177734375, -1.30859375, -0.041717529296875, 0.828125, -0.08697509765625, -4.03125, 0.38134765625, -1.2607421875, 0.1270751953125, 0.3486328125, 0.11212158203125, 0.8759765625, -1.607421875, 0.1610107421875, -0.08514404296875, 0.0038166046142578125, 1.0771484375, 1.302734375, -1.1533203125, 0.102294921875, 0.0843505859375, 0.68603515625, 0.8486328125, -1.8388671875, 0.53271484375, -0.505859375, 1.1513671875, -1.3583984375, 2.208984375, 1.173828125, -0.1864013671875, -0.04376220703125, 2.591796875, 1.302734375, 0.458984375, 0.4326171875, 0.88720703125, -0.317626953125, -0.7021484375, 1.0283203125, -0.5947265625, 0.5966796875, -0.0247344970703125, -0.0582275390625, 1.2548828125, -0.431396484375, -0.3896484375, 0.69189453125, 1.146484375, 1.412109375, 0.2425537109375, 1.0849609375, 1.515625, 0.436767578125, 1.9599609375, -0.005680084228515625, -0.08984375, 2.072265625, -0.10186767578125, 0.84033203125, -1.34765625, 0.114501953125, 2.115234375, -0.84521484375, 0.42822265625, -0.61572265625, 1.298828125, -0.2626953125, 0.139404296875, -0.449951171875, 0.039031982421875, -1.4833984375, 0.207275390625, 0.272705078125, -0.681640625, -1.3779296875, 1.4912109375, 0.061370849609375, -1.564453125, -1.859375, 1.158203125, 0.0002770423889160156, 1.21875, 0.0284576416015625, 0.20458984375, -0.007289886474609375, 1.6748046875, -0.7822265625, 0.349365234375, 0.51806640625, -0.79345703125, 0.188232421875, 0.22607421875, -2.369140625, 0.67138671875, -0.59521484375, -1.5029296875, 1.134765625, 0.464599609375, 0.2183837890625, 1.6162109375, -1.685546875, 0.88720703125, -0.791015625, -0.5146484375, -0.479248046875, 0.18310546875, 1.6025390625, -1.1865234375, -1.560546875, -0.8603515625, 0.35107421875, -0.1531982421875, -0.2078857421875, -0.09716796875, -1.3408203125, -0.25048828125, -1.09765625, -0.57177734375, -1.0908203125, -0.1031494140625, 0.09002685546875, 0.160888671875, -0.32080078125, 0.76611328125, -0.60302734375, 0.3017578125, 0.81005859375, 0.0345458984375, -0.27880859375, 0.10943603515625, -0.984375, -0.1611328125, 0.51513671875, -0.80810546875, 0.110595703125, 2.361328125, 1.017578125, -0.78955078125, 1.2314453125, -0.958984375, 1.0595703125, -0.033782958984375, 1.0322265625, 0.7001953125, -0.30224609375, 0.32373046875, -0.23681640625, -1.896484375, 0.494140625, -0.26123046875, -0.401611328125, -0.061065673828125, 0.400146484375, -0.1700439453125, 1.9970703125, 0.1942138671875, 0.51025390625, 0.242919921875, 0.462890625, -0.75830078125, 0.541015625, 1.8994140625, 1.859375, 0.0338134765625, -0.456787109375, 1.0341796875, -0.9462890625, -0.279052734375, 0.00217437744140625, -0.1934814453125, 1.2021484375, -1.2470703125, -0.314453125, 1.03515625, -1.59375, -0.08526611328125, -1.4716796875, 0.994140625, 0.09027099609375, -0.404296875, -0.394775390625, -1.1220703125, -1.134765625, 1.3408203125, -0.49267578125, 1.150390625, -0.30029296875, -0.64404296875, -0.23095703125, -0.95751953125, -0.5146484375, -0.26318359375, -0.4384765625, 0.88232421875, -0.09930419921875, 0.302734375, 2.44921875, 1.64453125, -0.8642578125, -1.2998046875, -0.0006356239318847656, -0.00617218017578125, -1.37890625, 0.28369140625, -1.0859375, 1.5673828125, 0.05255126953125, -1.9306640625, 1.8828125, -0.474853515625, -1.3037109375, -1.455078125, -0.87841796875, 0.490478515625, 0.87109375, -0.0653076171875, -0.8310546875, 0.7568359375, 0.19384765625, -0.65771484375, -0.57568359375, -0.59228515625, -0.27978515625, -0.68408203125, 1.09375, -0.73828125, 0.2396240234375, -0.01503753662109375, -0.71826171875, -0.281005859375, 0.256591796875, -0.060821533203125, 1.123046875, -1.2548828125, 2.30078125, 0.501953125, 0.399169921875, -0.0736083984375, -0.65087890625, 1.248046875, 0.326171875, 1.0029296875, 0.6142578125, -0.019866943359375, 1.4794921875, 0.55517578125, -1.275390625, -0.94189453125, 1.763671875, -0.25927734375, -0.3857421875, 0.429443359375, 0.30029296875, 0.1817626953125, 0.01313018798828125, 0.415771484375, 1.1337890625, -0.9541015625, -0.67578125, -1.3076171875, 0.6728515625, -0.406005859375, 0.0044403076171875, 0.293701171875, 0.58203125, 0.421875, -1.08203125, -1.1572265625, -0.87841796875, -1.5693359375, 0.3359375, -1.3447265625, -0.212890625, -0.002101898193359375, -0.94482421875, 1.6669921875, -0.1495361328125, 1.392578125, -0.2091064453125, -2.29296875, 0.1651611328125, -0.9013671875, -0.05548095703125, -0.55810546875, -0.92626953125, 0.31103515625, 1.4521484375, -1.8037109375, -0.69287109375, -0.5087890625, -0.2467041015625, 0.07208251953125, -0.7509765625, 0.7470703125, -0.359619140625, 1.509765625, 0.450927734375, -2.4296875, -3.416015625, -0.295166015625, 0.3466796875, -0.09527587890625, -0.287353515625, 0.294921875, -0.63720703125, -0.2310791015625, 1.4970703125, 1.0908203125, 1.3837890625, 0.378662109375, 0.1824951171875, -1.3603515625, -0.86962890625, -0.43017578125, 2.572265625, 0.98193359375, -2.134765625, 0.59716796875, 0.406982421875, -0.340576171875, -0.162109375, -1.0361328125, -1.40234375, 0.7939453125, 0.0264739990234375, 0.1314697265625, 1.580078125, 1.607421875, -1.5498046875, 2.013671875, -0.3623046875, -0.751953125, 0.373779296875, -0.1143798828125, 0.0858154296875, -0.139892578125, 1.1884765625, -0.58642578125, 0.521484375, -0.7353515625, -0.038330078125, 0.01192474365234375, -1.9169921875, 0.13330078125, -1.86328125, -1.6806640625, -0.310302734375, 1.654296875, 1.3896484375, 0.2332763671875, 0.93408203125, -1.3271484375, -0.5830078125, 0.1046142578125, -0.43115234375, 0.21337890625, -0.89306640625, -0.07763671875, 0.29443359375, 0.0196075439453125, -1.451171875, 0.98046875, -0.47900390625, 0.66064453125, 0.78466796875, -0.437255859375, -0.3583984375, 1.8955078125, -1.3125, 0.2381591796875, 0.77880859375, 1.158203125, -0.333984375, -0.89208984375, -0.77294921875, -0.71337890625, -0.1865234375, 0.20556640625, 1.28515625, 0.3232421875, -0.141357421875, 0.6201171875, 0.3271484375, 0.58544921875, -1.203125, -1.3037109375, -0.456298828125, 1.0517578125, 0.297607421875, -1.5146484375, -1.8759765625, 0.6025390625, -1.53125, 0.0338134765625, -0.2939453125, 0.41796875, -0.62255859375, -0.1517333984375, -0.95361328125, 1.5009765625, -0.2232666015625, 0.69970703125, -1.7119140625, 0.7255859375, -1.0712890625, -1.130859375, -1.3046875, 0.45263671875, 0.0623779296875, 0.359619140625, 0.339599609375, -0.80859375, -0.7890625, -0.07379150390625, -0.289306640625, -1.0, 0.6923828125, -0.70361328125, -0.60302734375, -0.034088134765625, -0.06536865234375, 1.2314453125, 0.1552734375, 1.0595703125, -0.470703125, -0.2222900390625, -0.51025390625, -0.7236328125, 0.8740234375, 1.048828125, -0.201416015625, -1.7392578125, 0.71337890625, -0.1617431640625, 1.0068359375, 0.043060302734375, 0.89404296875, -1.2802734375, 0.10809326171875, 0.31005859375, -0.6826171875, -1.8056640625, 1.3447265625, 0.431884765625, -1.5986328125, -0.2222900390625, 0.1910400390625, 0.60400390625, 0.5927734375, 1.0751953125, -0.73876953125, 0.671875, 0.2132568359375, 0.35498046875, -0.822265625, -1.4619140625, 0.149169921875, -1.2158203125, 0.962890625, -0.94140625, 0.92138671875, 0.865234375, -1.8798828125, -0.50634765625, 1.3974609375, -1.626953125, -0.49609375, 0.61767578125, -1.76171875, 0.60595703125, -0.9462890625, 0.176513671875, 0.0887451171875, 1.6591796875, -0.071044921875, -0.87744140625, 0.257568359375, 0.378173828125, -1.4140625, -0.8125, 0.1541748046875, 0.178466796875, -0.037261962890625, -0.82470703125, 1.1279296875, -0.385986328125, 1.5224609375, -0.283935546875, 0.2509765625, 0.26123046875, -2.90625, 0.303466796875, -1.984375, -0.413330078125, 0.006439208984375, 0.65478515625, 0.25634765625, 0.6962890625, 1.2001953125, -1.54296875, -2.822265625, -0.26025390625, -0.311279296875, 0.02008056640625, -0.495361328125, 1.671875, 0.9609375, -0.6953125, -0.50244140625, -0.1878662109375, 0.204833984375, -1.4208984375, 1.015625, 0.04345703125, -0.181396484375, 1.3193359375, 0.334716796875, -0.80029296875, 1.23828125, -0.183349609375, -1.568359375, -0.353271484375, -0.302978515625, -0.261962890625, -0.91552734375, -0.1480712890625, 1.060546875, 0.63671875, -0.04193115234375, 2.2578125, -1.3662109375, -0.1427001953125, 0.416015625, 1.14453125, -0.64111328125, 1.080078125, -0.322265625, 0.307861328125, -1.0703125, 0.396728515625, -0.212890625, 2.181640625, -0.78564453125, 0.61279296875, -0.425048828125, -0.91650390625, -0.56005859375, -0.37158203125, -1.2177734375, 1.0986328125, 0.285400390625, 0.24072265625, -0.6005859375, -0.45703125, 0.310791015625, 0.8046875, -1.041015625, -0.65283203125, 0.7939453125, -0.83056640625, 0.128662109375, -0.4384765625, -1.2958984375, 0.623046875, 0.465576171875, -0.84375, 0.73828125, -0.06390380859375, -0.51806640625, 0.356689453125, -1.1630859375, -1.0908203125, 2.03515625, 0.66015625, 0.50146484375, 0.40234375, 0.6953125, 0.18310546875, -0.7607421875, 0.496826171875, 0.73046875, -0.98193359375, 0.233642578125, 0.88818359375, -0.5849609375, -0.477294921875, 0.92138671875, -0.859375, 2.455078125, -0.29736328125, 0.7880859375, -1.2734375, 1.2119140625, -2.42578125, -0.34326171875, 2.701171875, 0.236328125, -0.15625, 0.1309814453125, -1.1767578125, 1.296875, 1.455078125, -0.2122802734375, -1.201171875, -0.85791015625, -0.6435546875, 1.7744140625, -0.2218017578125, 2.439453125, 0.1300048828125, 0.9345703125, 1.103515625, 0.425537109375, 0.346435546875, 0.93505859375, -0.89453125, -1.28515625, -1.349609375, -0.0296630859375, -0.044525146484375, -0.87109375, 0.1885986328125, 1.078125, -0.014495849609375, 1.37109375, -0.96484375, 0.14794921875, 0.6923828125, -0.375732421875, -1.0107421875, 0.243896484375, -2.0390625, 0.8515625, -0.89111328125, 0.2288818359375, -1.3251953125, -1.8046875, 2.021484375, -0.98681640625, 0.333984375, -0.54833984375, 0.2802734375, -0.5869140625, 1.095703125, -3.75, -1.82421875, 0.9765625, -1.44140625, 1.7548828125, 0.5283203125, -0.7861328125, 1.3076171875, -0.2379150390625, 0.431640625, -2.25390625, -1.4091796875, 0.323486328125, 0.9248046875, 1.8828125, 0.341064453125, -0.54443359375, -0.873046875, -0.09576416015625, 0.80322265625, 1.400390625, 0.5146484375, 0.1448974609375, -0.4873046875, 2.251953125, 0.234375, 0.68798828125, 1.615234375, 0.51953125, 1.447265625, 0.541015625, -1.2021484375, -2.01171875, 1.48046875, 1.0283203125, 1.0595703125, -1.2548828125, 0.1644287109375, -0.67138671875, -1.1015625, -0.10699462890625, 1.7890625, -0.62451171875, 1.2978515625, -1.240234375, 0.2354736328125, -0.30712890625, -0.478515625, -1.74609375, 0.2440185546875, -1.587890625, -0.33837890625, -0.87548828125, -0.239013671875, -0.418701171875, -0.79248046875, 0.87548828125, 1.4404296875, 1.720703125, 0.169921875, 0.4091796875, -0.288818359375, -1.681640625, 0.291748046875, -0.5654296875, 0.7578125, 1.0615234375, -0.3759765625, -1.3212890625, -0.2425537109375, -0.4443359375, -0.751953125, 1.234375, 0.09423828125, 0.68603515625, -0.0254669189453125, -0.230224609375, -1.91015625, -0.01343536376953125, 0.61474609375, -1.6953125, -1.015625, -1.1376953125, -0.47021484375, 1.5986328125, 1.9111328125, -0.40869140625, -1.30859375, -0.1285400390625, -0.513671875, 0.350830078125, -0.85107421875, -0.35498046875, -0.411376953125, 0.03131103515625, 1.4072265625, 0.259521484375, 0.258056640625, 1.2021484375, 0.16552734375, -0.97314453125, 1.5302734375, 0.69384765625, -1.3154296875, 0.42919921875, 0.57763671875, -0.388427734375, -0.45166015625, -1.1708984375, 0.3427734375, 1.875, 1.392578125, -0.173583984375, 0.04541015625, 0.03106689453125, -0.6064453125, 0.0225677490234375, 0.475830078125, 1.150390625, -0.0618896484375, 0.68408203125, 1.224609375, 2.220703125, 1.8583984375, 0.34765625, 1.3017578125, 0.08990478515625, -0.1737060546875, -1.3427734375, -1.14453125, 0.2802734375, -2.931640625, 2.61328125, 2.255859375, 0.56201171875, -1.076171875, 1.2314453125, -0.1275634765625, 0.048370361328125, -0.38525390625, 1.3974609375, -0.833984375, 0.2435302734375, 0.7294921875, -0.383056640625, -0.1304931640625, 0.65283203125, 0.619140625, -1.22265625, 2.484375, 0.168701171875, -0.89794921875, -1.7744140625, 0.859375, -0.436767578125, -0.06256103515625, -0.62158203125, -0.343505859375, 0.08160400390625, 0.7763671875, 0.076171875, -1.3076171875, -0.23486328125, 0.75439453125, -0.6875, -1.35546875, 0.307373046875, -0.4033203125, 0.09954833984375, 1.29296875, 0.463623046875, 0.9443359375, 0.583984375, -0.473388671875, -1.2314453125, 0.275146484375, 0.07269287109375, -0.0919189453125, 0.80517578125, 1.4599609375, 0.047576904296875, 0.314208984375, -0.279296875, 1.5927734375, -1.34375, 0.64599609375, 0.4716796875, 0.11639404296875, 1.4755859375, -1.2626953125, -0.6435546875, 0.59423828125, -0.22802734375, 0.5576171875, 0.150634765625, 1.0322265625, -0.5498046875, 0.1312255859375, -0.18359375, -0.321533203125, 2.640625, -0.5927734375, -0.0268402099609375, 0.3876953125, 0.6513671875, -0.308349609375, -0.37646484375, 0.859375, 0.7431640625, 0.88916015625, -1.388671875, 0.962890625, 0.81640625, 1.2314453125, 1.1728515625, 1.41015625, -1.5791015625, -0.6806640625, -0.77294921875, -1.23046875, -1.53125, 0.2447509765625, -1.6435546875, 1.0927734375, -0.1507568359375, -0.447998046875, 2.0546875, 0.92333984375, 0.1697998046875, 0.218994140625, -0.6611328125, 0.89306640625, -1.6845703125, 0.6240234375, -1.7021484375, 0.63525390625, -0.171630859375, 0.265380859375, -0.39453125, 0.98681640625, -0.187744140625, 1.732421875, -2.615234375, 0.69189453125, -0.2205810546875, 1.841796875, 1.63671875, -1.46484375, -0.303955078125, -0.1964111328125, 0.60888671875, -0.2978515625, 0.54248046875, -0.4228515625, 0.1715087890625, 0.56689453125, -0.51220703125, 0.0836181640625, -0.2093505859375, -2.078125, 0.56494140625, -1.8388671875, -1.00390625, 1.3310546875, -0.931640625, 1.5888671875, -0.50439453125, -1.4892578125, -0.35888671875, 0.0178070068359375, 0.16748046875, -0.8759765625, -1.45703125, -0.43212890625, -0.470703125, -0.125, -1.2607421875, -0.322998046875, -0.31591796875, 0.319091796875, -0.6201171875, 0.51806640625, -0.49365234375, 0.302001953125, 0.72509765625, -1.033203125, -0.7705078125, -0.787109375, 1.541015625, 0.986328125, -1.009765625, -0.2548828125, -0.7783203125, -0.1810302734375, 0.297119140625, -0.3310546875, 0.4921875, 0.10968017578125, 0.5341796875, -0.60693359375, -0.169677734375, 0.89599609375, -0.1622314453125, 0.06549072265625, 0.91357421875, -0.56982421875, 0.9736328125, 0.583984375, 0.0208892822265625, -0.77978515625, 0.98095703125, -1.6220703125, -2.453125, 0.49462890625, 0.473388671875, 0.79248046875, 0.9072265625, -2.71484375, 0.2486572265625, -1.1572265625, -1.462890625, -1.3291015625, 1.240234375, 0.68310546875, 0.548828125, -0.77197265625, -1.703125, -1.1650390625, -0.208984375, 1.2578125, 1.4111328125, 1.6845703125, -1.0146484375, 0.2900390625, -1.5205078125, 1.326171875, 0.1529541015625, 0.9951171875, 2.001953125, 1.07421875, 1.697265625, -1.05078125, 0.3916015625, 0.4560546875, -0.314697265625, -0.432861328125, 0.200439453125, 1.0234375, -1.171875, 0.56005859375, 3.703125, -0.60888671875, 2.23046875, 0.6533203125, -1.8583984375, -1.5791015625, 0.84521484375, -0.51025390625, 0.258056640625, -0.2021484375, 0.471923828125, 0.3203125, -0.65185546875, 0.6650390625, 1.7373046875, 1.037109375, 0.943359375, 0.427490234375, 0.19677734375, -0.5087890625, 1.4345703125, -0.6650390625, 0.0909423828125, -0.48095703125, -2.904296875, -0.64892578125, -0.3671875, 1.119140625, 1.2041015625, -1.38671875, -2.31640625, -0.4091796875, -1.142578125, -0.9443359375, 0.386474609375, -0.5908203125, -0.25341796875, 0.84814453125, 0.56298828125, 0.114013671875, 1.1357421875, -0.7275390625, -1.173828125, 1.275390625, 1.322265625, 0.429931640625, -0.40576171875, 1.306640625, -0.7099609375, 1.2939453125, -0.1695556640625, 1.4814453125, 0.794921875, -0.609375, 0.17138671875, -2.115234375, -0.1859130859375, 2.3125, 1.0263671875, -1.1298828125, -0.41162109375, 1.2421875, 0.69921875, -1.1630859375, -0.262939453125, -1.2685546875, -1.9287109375, 0.9619140625, 2.51171875, 0.54833984375, 0.57080078125, 0.2156982421875, -0.40673828125, 0.1849365234375, 1.0927734375, -0.2626953125, 1.884765625, -1.4853515625, -0.99951171875, 0.3427734375, -0.60595703125, -0.482421875, 0.117431640625, -1.75, -0.1639404296875, -0.82958984375, 1.05859375, 0.572265625, -0.70703125, -0.7939453125, -0.203857421875, 0.17919921875, -1.048828125, 0.11712646484375, 0.1702880859375, -1.7294921875, 0.4365234375, 1.94140625, -0.1322021484375, -0.321533203125, -0.2666015625, -0.277587890625, -0.99951171875, 1.2568359375, 0.1783447265625, 1.462890625, 1.1787109375, -0.1495361328125, -0.1676025390625, 1.32421875, 0.92236328125, 2.490234375, 0.43212890625, -1.0654296875, 0.7021484375, -2.619140625, -1.6220703125, -1.5029296875, -0.278564453125, 1.080078125, -0.4091796875, -2.55859375, -0.0041046142578125, 1.3564453125, 1.615234375, -0.0570068359375, -1.3857421875, 0.6005859375, 0.2337646484375, -0.47900390625, 0.6787109375, -1.63671875, 0.1553955078125, -0.5712890625, -0.93359375, -1.46875, -0.365478515625, 0.330322265625, 0.2861328125, 0.98193359375, -0.5048828125, 0.623046875, -1.322265625, -0.751953125, -1.1748046875, 0.7470703125, -0.024627685546875, 0.85302734375, -0.229248046875, -1.7490234375, -0.1961669921875, 0.8408203125, 1.2373046875, -0.650390625, 1.9208984375, 0.9443359375, 0.1925048828125, -0.82177734375, -0.77587890625, 1.7578125, -0.3818359375, 0.2266845703125, -0.383056640625, 0.58447265625, -0.62744140625, -0.306396484375, 0.9013671875, 0.9521484375, -0.11737060546875, 0.162353515625, -0.77001953125, 1.556640625, 1.4423828125, 2.89453125, 1.5771484375, 0.552734375, 0.74560546875, -0.157470703125, 0.07647705078125, 0.67236328125, -0.77197265625, -0.693359375, 0.06341552734375, -0.4091796875, -1.17578125, -0.468017578125, 0.015045166015625, 1.4765625, -0.237548828125, 0.509765625, 0.08978271484375, -1.3173828125, 0.53125, -0.349365234375, -1.1494140625, 0.372802734375, 0.277099609375, -2.017578125, -0.367431640625, -0.82275390625, 1.55078125, 1.7919921875, 0.41650390625, -0.221923828125, -0.63427734375, 1.830078125, 0.73974609375, 2.43359375, -0.317138671875, 1.091796875, 0.2269287109375, 0.48876953125, -0.36572265625, -1.6455078125, 0.332275390625, -0.457275390625, 2.486328125, 0.10980224609375, 0.1690673828125, -0.465576171875, -0.0044403076171875, -0.97119140625, 1.7841796875, 0.236328125, 0.0019397735595703125, -0.0184478759765625, 0.7509765625, 0.9521484375, -0.08831787109375, -0.318603515625, -1.8642578125, 2.154296875, 0.8671875, 0.092529296875, -0.85546875, 0.45068359375, 0.25048828125, 0.59716796875, -1.7607421875, 0.6748046875, 0.5498046875, 0.3720703125, 0.6923828125, 1.1064453125, -0.51904296875, 1.0908203125, 0.3935546875, -2.1328125, 2.048828125, -2.099609375, 2.04296875, -2.173828125, -0.4619140625, 0.71875, -0.359130859375, 1.7880859375, 0.5986328125, -1.9111328125, 1.3076171875, -0.63623046875, 0.5322265625, -0.7626953125, -1.22265625, -0.884765625, 0.01343536376953125, -0.619140625, 0.10797119140625, 0.697265625, -1.4775390625, 0.280029296875, -0.30712890625, 0.97509765625, 0.99169921875, -1.189453125, 0.67919921875, -1.1826171875, 0.51904296875, -0.08050537109375, 1.5986328125, -0.285888671875, -0.77685546875, -0.51123046875, -0.9462890625, 3.037109375, -0.0711669921875, -0.031280517578125, -1.7900390625, -1.841796875, -0.6572265625, -0.7470703125, 0.716796875, -1.703125, 1.3134765625, -0.94287109375, -1.1982421875, -0.00963592529296875, 1.005859375, -0.1480712890625, -1.2421875, -0.55419921875, 0.310791015625, -2.001953125, -0.904296875, 0.40380859375, -1.3212890625, 1.0419921875, -1.5576171875, 0.144775390625, -0.56591796875, -0.251953125, 0.319580078125, 0.52587890625, 1.732421875, -2.279296875, -0.8994140625, -1.521484375, 0.490966796875, 1.869140625, 0.90625, 0.0982666015625, -0.82275390625, -0.54931640625, -1.1904296875, 1.8740234375, 0.80029296875, 1.7548828125, -1.2939453125, 1.16015625, -1.90234375, 1.28125, -0.90625, -1.14453125, -0.490234375, 0.72705078125, -0.5595703125, -0.272705078125, -1.0703125, 0.85791015625, -0.7587890625, -0.462646484375, 0.0789794921875, 0.442626953125, -0.92138671875, -1.17578125, 0.1961669921875, 0.12127685546875, -1.01171875, 2.005859375, 0.435302734375, -0.432861328125, -0.6181640625, -1.1591796875, 0.45068359375, 0.83935546875, 0.513671875, 1.423828125, 0.6025390625, -0.76806640625, 1.72265625, -0.324951171875, -0.79052734375, -0.419677734375, -0.90283203125, -0.521484375, 0.1396484375, -1.30859375, -1.4912109375, 1.728515625, -1.4833984375, 1.7099609375, -1.83984375, -1.8310546875, -1.3828125, -0.79052734375, 0.354248046875, 0.7119140625, 0.1629638671875, 1.52734375, -0.55517578125, 1.40234375, -0.83251953125, 0.2330322265625, 0.87939453125, 0.306396484375, 0.15380859375, 0.4072265625, 0.5830078125, -0.8046875, -0.337158203125, -2.2421875, 0.281005859375, -0.29833984375, 1.333984375, -1.390625, -0.58984375, 1.04296875, -2.015625, 0.420166015625, -1.75, 1.5703125, -1.763671875, -1.626953125, -0.35986328125, 1.263671875, -0.35693359375, -2.05078125, 0.346923828125, -2.4609375, -1.607421875, -1.619140625, 0.046051025390625, 1.3896484375, 0.305908203125, -0.8779296875, -0.2208251953125, -0.2220458984375, -0.0156402587890625, 2.1484375, -0.140625, -2.234375, -1.7919921875, 0.3818359375, -0.97119140625, 0.708984375, 0.0626220703125, 0.272216796875, 2.564453125, 0.6875, 0.63623046875, -1.310546875, 0.67724609375, 2.263671875, -0.61376953125, -0.83251953125, 0.168701171875, -0.84521484375, 1.31640625, -1.46875, 0.12213134765625, -0.2064208984375, 1.3369140625, 0.5263671875, -1.6708984375, 0.50048828125, 0.6259765625, 0.5576171875, -0.288818359375, 0.74658203125, 0.2254638671875, -0.44873046875, -1.22265625, -2.6171875, -1.3525390625, -0.81201171875, 1.0732421875, -1.5126953125, -1.27734375, -0.50341796875, -1.384765625, 0.443115234375, -0.23193359375, -0.268798828125, 0.052764892578125, 0.33544921875, -0.041778564453125, 0.1932373046875, -0.53759765625, -0.51806640625, -0.08990478515625, -0.6259765625, 0.338134765625, -0.61474609375, -0.367431640625, -1.1015625, 0.6796875, 0.435546875, -1.080078125, 0.53173828125, -0.271728515625, 0.69580078125, 1.345703125, 0.560546875, 0.01549530029296875, 0.5458984375, 0.0863037109375, -0.55810546875, 0.91259765625, -0.266357421875, 0.2685546875, 0.16455078125, -1.6533203125, -0.291748046875, -0.873046875, 0.68994140625, 0.23681640625, -0.1795654296875, -0.424072265625, -0.12493896484375, 0.03363037109375, 0.482177734375, -0.58984375, -0.01319122314453125, 0.84521484375, -0.430419921875, 0.57666015625, 0.29150390625, -1.298828125, 1.0009765625, 2.21875, 1.3427734375, -0.2103271484375, 0.283203125, -0.029266357421875, -0.345458984375, 1.0751953125, 0.9267578125, -0.385009765625, -0.302001953125, -0.88232421875, -0.2056884765625, -1.3359375, 0.75244140625, 0.07159423828125, 0.313720703125, 0.5556640625, 0.2509765625, -0.06353759765625, -0.935546875, 2.11328125, -0.403564453125, -1.1904296875, 0.10565185546875, 1.6689453125, 0.71728515625, 0.90478515625, -1.0185546875, -1.9755859375, 0.46923828125, -1.2998046875, 1.220703125, -0.323486328125, -0.58447265625, -0.135498046875, -1.4482421875, -1.34375, -1.4365234375, 0.1552734375, -0.2227783203125, 1.9150390625, 1.19140625, -1.001953125, 0.28173828125, -1.201171875, 0.77197265625, -0.288330078125, -1.09375, -0.6484375, 0.2100830078125, 0.52978515625, -1.119140625, -0.06427001953125, 0.3681640625, 0.9052734375, 0.8076171875, 0.5791015625, 0.250244140625, 0.64990234375, 0.56298828125, 0.18310546875, 0.10833740234375, -0.44580078125, 0.66748046875, 0.77880859375, 1.1875, 0.6875, 0.09124755859375, -1.3427734375, -0.5302734375, -0.8310546875, 0.07745361328125, 0.449951171875, 0.7490234375, -0.50732421875, -0.98486328125, -0.26171875, 0.97705078125, -0.338623046875, 2.8984375, -0.8037109375, -2.45703125, -0.53759765625, -0.441162109375, -2.341796875, 0.369384765625, -0.448974609375, 1.0615234375, -0.267822265625, -1.5712890625, -0.478271484375, -0.0203399658203125, 1.4013671875, 0.117431640625, -1.435546875, -1.1572265625, 0.8466796875, 0.75048828125, 0.394775390625, -0.2158203125, -0.2410888671875, 0.51123046875, -0.365966796875, -0.3525390625, -1.0947265625, -1.8720703125, -0.7685546875, -0.2177734375, -0.399658203125, -0.23779296875, -0.794921875, 2.806640625, -0.90185546875, 0.70263671875, -0.85107421875, -0.59716796875, 0.2049560546875, 0.89990234375, 2.509765625, -0.444580078125, 0.056610107421875, 0.92578125, 1.1171875, -0.1380615234375, -2.8828125, 0.251708984375, -1.6240234375, 0.376708984375, -0.397705078125, 0.0028934478759765625, -0.70068359375, -0.84521484375, 0.38427734375, 1.1728515625, 0.6015625, 0.489013671875, -0.900390625, -1.1201171875, -0.4150390625, -0.09735107421875, 0.84912109375, -1.6826171875, 1.0947265625, -1.79296875, -1.162109375, 2.33984375, -2.259765625, 0.30615234375, 0.06182861328125, 0.62158203125, -0.94921875, 1.544921875, 0.08074951171875, 0.9638671875, 1.181640625, 0.57470703125, -0.62744140625, -1.955078125, 1.2958984375, -0.129150390625, -1.294921875, -1.5986328125, 0.2198486328125, 1.0732421875, -0.349609375, -0.25439453125, -0.102783203125, -0.08331298828125, -2.126953125, -1.248046875, -0.701171875, -1.16796875, -0.240234375, 1.33984375, 2.05078125, -0.435791015625, 0.09326171875, -0.11517333984375, 0.150146484375, 0.9677734375, 0.80908203125, -1.9189453125, 0.8857421875, -0.060882568359375, -0.426513671875, -0.048736572265625, -0.6669921875, -0.962890625, 0.66162109375, -1.2021484375, -0.35595703125, -1.947265625, 0.447021484375, 0.283935546875, -0.298828125, 0.394287109375, -0.4287109375, -2.17578125, 0.62744140625, 0.112548828125, 2.998046875, 0.0372314453125, 1.8203125, 0.9921875, 1.0146484375, -0.37890625, -0.86767578125, -1.1474609375, -0.301025390625, 2.2734375, 0.072509765625, 1.48046875, -0.48046875, 0.82421875, 0.27978515625, 1.6669921875, 2.962890625, 1.1435546875, -0.4873046875, -0.1353759765625, 0.93798828125, 0.6005859375, -1.009765625, -0.06292724609375, -0.23779296875, -1.6357421875, 0.328857421875, 0.6396484375, 0.951171875, -2.0078125, 1.86328125, -1.25, -0.5185546875, -1.5849609375, -0.297607421875, 0.046875, 0.79052734375, -0.08831787109375, -1.7802734375, 0.92724609375, 0.051422119140625, -1.625, 1.2138671875, 0.03839111328125, -0.37060546875, -0.142822265625, 0.0667724609375, -0.57958984375, -0.72607421875, 0.740234375, -0.3974609375, -0.97900390625, 0.05670166015625, 0.4404296875, 0.9462890625, -3.212890625, 0.72021484375, -0.640625, 0.09661865234375, 0.474609375, 0.373291015625, -0.10369873046875, 1.9599609375, 2.294921875, 0.65283203125, -0.93310546875, -1.2373046875, 1.2646484375, -1.9375, 0.055999755859375, 0.1839599609375, 0.151611328125, -0.156494140625, -0.1429443359375, -0.0307769775390625, -0.029205322265625, 0.5869140625, -0.40283203125, 0.3955078125, 0.396728515625, 0.3203125, 0.61328125, -1.2529296875, -0.043975830078125, 0.07354736328125, 0.75927734375, -1.41796875, -1.12109375, -0.321044921875, -0.70947265625, -0.1640625, -2.009765625, 0.8623046875, -0.2900390625, 0.8828125, -0.6826171875, -0.461669921875, -1.9140625, -0.284423828125, -0.48876953125, 0.1962890625, 1.0087890625, -0.8271484375, -0.176025390625, -1.4033203125, -0.77099609375, -0.183349609375, 0.654296875, -0.4130859375, -0.96044921875, -0.052581787109375, 2.0390625, -0.1619873046875, 1.4072265625, 0.96533203125, -0.021331787109375, 0.99853515625, -0.38330078125, -0.363037109375, 0.47119140625, 0.193359375, 2.337890625, 0.7490234375, -1.18359375, 0.197998046875, -0.916015625, -0.337158203125, -0.282470703125, -1.0458984375, 0.5849609375, -0.26416015625, 0.12744140625, 1.1630859375, 0.76123046875, 0.142822265625, 1.306640625, -0.814453125, 0.01001739501953125, -0.58984375, 0.0423583984375, -2.244140625, 1.0302734375, 0.1505126953125, -1.037109375, -0.94384765625, -0.65185546875, 0.80810546875, 0.7001953125, 0.09429931640625, -0.9208984375, -0.055267333984375, -1.228515625, 0.39501953125, -0.474853515625, -0.80859375, 0.8349609375, -1.9638671875, 0.9189453125, -0.51904296875, 0.24560546875, -0.5986328125, -0.1702880859375, 0.195556640625, -1.119140625, -0.1068115234375, -1.6162109375, 0.0411376953125, 0.63525390625, 1.9111328125, -2.1171875, -0.277587890625, 0.49951171875, -0.6748046875, -0.76025390625, 0.0310821533203125, -0.89306640625, 0.491455078125, 1.1416015625, 2.12890625, 0.07220458984375, 1.1640625, 0.11749267578125, -0.036224365234375, 0.1796875, -1.05078125, 0.82861328125, -0.2313232421875, -0.7138671875, -0.53369140625, 0.5087890625, 0.086669921875, 0.429443359375, -0.376220703125, -0.28759765625, 0.43408203125, -1.1103515625, -0.0294647216796875, -1.271484375, -0.28759765625, 0.05450439453125, 1.02734375, 1.625, -0.352294921875, 0.0894775390625, 1.271484375, 0.10540771484375, -1.123046875, -2.236328125, 0.470458984375, -0.10211181640625, -0.50732421875, 0.87255859375, 0.61083984375, -0.1629638671875, 0.19287109375, 1.693359375, 0.481689453125, 0.3056640625, 0.62548828125, 0.352294921875, 0.08209228515625, -0.270263671875, 0.2127685546875, -0.50439453125, 0.2083740234375, -0.5927734375, -0.92822265625, -0.55078125, 1.3876953125, -0.67138671875, 0.178466796875, -0.5087890625, 0.059051513671875, 0.54296875, -0.529296875, -0.048004150390625, -1.408203125, -0.01275634765625, 0.189697265625, 1.5400390625, -0.48388671875, 0.9755859375, 0.2298583984375, 1.4970703125, 0.00753021240234375, 2.330078125, -0.9560546875, 0.466552734375, -1.0615234375, 0.63427734375, -0.162109375, 0.1614990234375, -0.397216796875, 2.0, -1.546875, -0.1666259765625, -0.031646728515625, 0.11083984375, -0.955078125, 0.8701171875, -1.9462890625, 0.364990234375, -1.4794921875, -0.67041015625, 1.3427734375, -0.023834228515625, 0.01323699951171875, -0.52001953125, -0.2447509765625, 0.305908203125, -1.220703125, 1.5048828125, 0.56494140625, 0.6181640625, -0.2327880859375, 1.8505859375, 0.3642578125, 0.0986328125, -1.3818359375, -1.3681640625, -0.433837890625, -0.095703125, -0.88134765625, -1.3232421875, -0.2291259765625, 1.0185546875, 0.77734375, -1.0205078125, -0.0345458984375, 0.89599609375, 0.87060546875, 1.3310546875, -0.84326171875, -0.016082763671875, -2.130859375, 1.484375, -1.359375, -1.0712890625, 1.07421875, 1.525390625, -0.0499267578125, 0.50439453125, 1.2861328125, -0.53271484375, -1.189453125, -0.54345703125, -0.59130859375, -0.1234130859375, -1.3701171875, 1.640625, -1.4248046875, -0.0130157470703125, -1.15234375, 0.334716796875, -1.0341796875, -0.53857421875, 1.298828125, 2.697265625, -0.73095703125, 0.26708984375, -1.0478515625, 1.2783203125, 0.189697265625, 0.1094970703125, 0.403076171875, -1.966796875, -0.1494140625, -0.46435546875, 0.2744140625, -0.378662109375, -0.9638671875, -0.2373046875, -0.79443359375, -2.42578125, 0.73779296875, -1.0478515625, 0.247314453125, -0.72119140625, 0.042572021484375, 0.74755859375, -0.7197265625, -1.77734375, -0.72216796875, 0.476318359375, -1.6630859375, -1.7021484375, 0.395263671875, 0.56591796875, -0.33740234375, 1.2099609375, -0.015045166015625, -0.40380859375, -0.96923828125, 1.0087890625, -0.05255126953125, -0.732421875, -0.28857421875, 1.1494140625, 1.203125, 0.12091064453125, 0.2144775390625, -0.434326171875, -1.6357421875, -0.062347412109375, -0.52197265625, 0.1142578125, 0.9619140625, 0.47314453125, 1.2158203125, 0.06591796875, -0.58740234375, -0.7314453125, 0.253173828125, -0.53466796875, 0.39501953125, 0.99609375, -0.95263671875, 0.10772705078125, 1.22265625, -0.0214996337890625, -0.453369140625, -1.3232421875, 0.467529296875, -0.205322265625, 0.043182373046875, -1.6240234375, 0.372802734375, 1.0703125, -1.6904296875, -0.90673828125, 0.68017578125, -0.62158203125, -0.020660400390625, 0.40869140625, -0.6806640625, -0.939453125, 1.0576171875, 0.29248046875, 1.4267578125, 1.2802734375, -0.689453125, -1.7265625, 0.82421875, 0.11322021484375, 0.419921875, 0.5888671875, 0.74462890625, -0.252197265625, -1.15234375, 0.390380859375, 1.1435546875, -0.27294921875, -1.330078125, -0.68408203125, -0.2176513671875, 0.43408203125, -1.91796875, -0.2919921875, -1.7177734375, 0.424072265625, 0.68505859375, -0.351318359375, 0.84423828125, -0.498291015625, -1.5546875, -0.55126953125, 1.94921875, 0.6728515625, 0.119873046875, 0.80419921875, -0.49560546875, 0.059814453125, 0.013519287109375, 0.01013946533203125, 0.3515625, -1.1875, -2.341796875, -0.45947265625, -0.1180419921875, 0.240966796875, 0.95654296875, 0.763671875, 2.478515625, -0.044677734375, -1.240234375, 1.08984375, -0.442626953125, -0.43212890625, 0.43408203125, -0.1446533203125, -0.7392578125, 1.0751953125, -1.5595703125, 0.7744140625, 0.4326171875, -1.087890625, -0.419921875, 0.40771484375, 1.23828125, -0.98828125, 0.328369140625, -0.52734375, -0.255859375, 0.482421875, 0.56494140625, -0.11669921875, 0.333740234375, -0.258544921875, 0.0771484375, 1.439453125, 0.474365234375, -1.3818359375, -0.26318359375, -1.2783203125, 0.380126953125, 0.6884765625, 0.394775390625, -0.98291015625, 0.080810546875, -0.2496337890625, -0.95654296875, 1.09375, 0.0156402587890625, -1.119140625, -0.55810546875, 0.63916015625, 0.303466796875, -1.353515625, -0.263916015625, -0.42041015625, -0.5146484375, 2.044921875, 0.264892578125, 0.55615234375, 1.4130859375, -0.728515625, -1.84375, 1.826171875, -1.603515625, 0.33837890625, 1.93359375, 1.3564453125, -0.1329345703125, 0.173583984375, -0.92919921875, 0.94775390625, -0.30126953125, 1.2822265625, -0.68505859375, -0.04766845703125, 1.291015625, -0.2138671875, 0.098876953125, 0.25537109375, 0.2198486328125, 0.78662109375, 0.658203125, -0.367919921875, -1.89453125, -0.92431640625, 1.9404296875, -0.330810546875, 1.4228515625, -1.1337890625, 0.01556396484375, 1.7744140625, -0.208740234375, -0.3388671875, 0.82275390625, 2.62890625, -1.568359375, 0.122314453125, 0.97802734375, -0.6650390625, -0.324951171875, -0.137939453125, 0.20458984375, 0.52001953125, 0.80029296875, 0.79052734375, -1.3115234375, -0.6728515625, 2.1015625, 0.8662109375, 0.95556640625, 2.20703125, 0.67138671875, 0.385009765625, -0.07794189453125, -0.8779296875, -0.10821533203125, -0.3017578125, -0.038482666015625, 0.79296875, -0.30859375, -2.25390625, -0.1656494140625, -0.59521484375, 1.2041015625, -0.1737060546875, 0.80615234375, -0.494140625, 0.00951385498046875, 0.67431640625, -0.045379638671875, 0.84765625, 0.177490234375, -0.21533203125, 1.23046875, -0.27783203125, 1.0087890625, -0.92578125, 0.8623046875, -1.333984375, 0.5263671875, 0.333984375, 0.2646484375, -0.7841796875, -1.388671875, 1.2314453125, 0.396484375, -0.625, 0.460693359375, 1.306640625, 0.1634521484375, 0.5146484375, 1.5712890625, -2.822265625, -0.1378173828125, 1.5517578125, 0.798828125, 0.62939453125, -0.6728515625, 0.148193359375, -0.40234375, 0.71728515625, 0.314697265625, 1.6923828125, -0.6630859375, -0.3125, -1.0, 0.43798828125, 0.2054443359375, 0.92236328125, 0.744140625, -0.038970947265625, 1.478515625, -0.91650390625, 0.11865234375, 1.0859375, -0.87158203125, 1.650390625, -1.298828125, 0.70361328125, -0.07476806640625, -0.54345703125, -0.982421875, 2.044921875, 0.50537109375, 0.352783203125, 0.533203125, 0.208740234375, 0.19482421875, 0.269775390625, 1.9609375, -0.153564453125, -0.5986328125, 0.373291015625, 0.765625, 1.5595703125, -0.421630859375, -0.42431640625, -0.51806640625, 0.22802734375, 0.0765380859375, -1.234375, 1.9423828125, 0.2396240234375, 0.319580078125, 0.712890625, 1.025390625, -1.455078125, -0.341064453125, -0.71484375, 0.4150390625, -0.421142578125, -0.55029296875, 0.8486328125, -0.03961181640625, 1.8466796875, 0.85107421875, -0.31884765625, -0.84619140625, -1.0810546875, 0.52490234375, 1.1201171875, -1.046875, -1.0634765625, -0.6201171875, 0.0892333984375, 0.07208251953125, 0.3291015625, -0.334716796875, -1.9306640625, -0.07598876953125, -1.294921875, 0.0704345703125, 0.408935546875, 0.19873046875, -0.083740234375, 0.27490234375, 1.029296875, -0.73388671875, -1.5947265625, 1.015625, 0.5595703125, -0.12744140625, 0.1622314453125, 0.59716796875, -0.80859375, 0.92822265625, 1.77734375, -0.52392578125, -0.262939453125, 1.03125, 1.7275390625, -0.78125, 0.412353515625, 0.84228515625, 0.1558837890625, 1.41015625, 0.736328125, 1.1240234375, 1.4599609375, 0.91845703125, -0.426025390625, -0.11566162109375, -0.06512451171875, -0.463623046875, -0.330322265625, -2.4453125, 0.276123046875, -1.484375, 2.384765625, -1.8125, -0.740234375, -0.8916015625, -1.525390625, -0.36181640625, -0.91455078125, -0.1207275390625, 0.986328125, -1.142578125, 0.319580078125, -1.1806640625, 0.38427734375, -0.285400390625, 0.2203369140625, -1.841796875, -0.05230712890625, 1.6796875, 1.080078125, -0.79443359375, -1.1767578125, 1.55859375, -1.4541015625, -0.43798828125, -0.3916015625, 0.5615234375, -1.4677734375, 0.84423828125, 0.4921875, -0.35791015625, -1.947265625, 0.9443359375, -0.247802734375, 1.0078125, -0.66455078125, 0.6630859375, 0.94921875, 0.9033203125, 0.140869140625, -0.66162109375, -0.476318359375, 0.984375, -0.07171630859375, -0.344482421875, 0.325927734375, -0.1832275390625, -0.73046875, 0.186767578125, 0.3193359375, -1.1103515625, 0.58203125, -1.0712890625, -0.884765625, 0.25, 0.151123046875, 1.4091796875, 0.398681640625, 0.2310791015625, 0.60546875, 0.947265625, -0.96240234375, -0.80712890625, 2.337890625, -0.251708984375, 0.2413330078125, 0.080810546875, -1.625, 0.0806884765625, -0.3173828125, 2.224609375, 0.51220703125, 0.25, -1.029296875, -0.10699462890625, 0.200927734375, 0.398193359375, -0.4853515625, 0.66845703125, 0.414306640625, 2.42578125, 0.09051513671875, 0.23974609375, 0.1826171875, 0.328369140625, 0.1934814453125, -0.344970703125, -0.1148681640625, -0.7236328125, 0.262939453125, -1.9697265625, -0.20458984375, -0.83642578125, -0.47412109375, -0.1651611328125, -1.2177734375, 0.09637451171875, 0.208984375, 0.05572509765625, -0.381103515625, 0.7802734375, -0.96142578125, -0.85400390625, 0.25439453125, 1.1142578125, 0.37158203125, -0.83837890625, -1.90625, -0.64111328125, -0.45068359375, -0.939453125, -1.330078125, 0.275390625, -0.437744140625, -0.1466064453125, -0.595703125, -1.6181640625, 0.3212890625, 1.1640625, -1.158203125, 0.1446533203125, -1.099609375, -0.349609375, 1.359375, 0.5498046875, 0.5439453125, 0.568359375, 0.55322265625, 0.255859375, -1.275390625, -0.356201171875, 1.013671875, -0.7578125, 0.65185546875, -0.76416015625, -0.419677734375, 0.427734375, -0.65673828125, 0.66845703125, -0.3447265625, -1.234375, -0.78662109375, 0.26025390625, -0.9130859375, 0.19873046875, -1.2626953125, -0.291748046875, -0.59228515625, -1.166015625, -0.876953125, -0.34765625, 0.36083984375, -1.1689453125, 0.42529296875, -0.67919921875, 1.439453125, 1.2724609375, -0.202880859375, 0.5654296875, -0.2376708984375, 0.12939453125, 0.089111328125, 0.80078125, -0.8369140625, -0.062103271484375, 0.79052734375, 0.5673828125, -0.058685302734375, 0.1302490234375, -0.1990966796875, -0.267578125, 0.02117919921875, -0.5302734375, 1.2080078125, -0.3779296875, -0.1336669921875, 0.45068359375, 1.8076171875, 0.74169921875, -1.20703125, 0.7080078125, -0.327880859375, -0.46826171875, -2.0703125, -0.08819580078125, 1.6376953125, 2.35546875, -0.98291015625, -1.451171875, -1.17578125, 0.199462890625, 0.966796875, 0.96630859375, -0.050872802734375, -0.99951171875, 0.1702880859375, 1.8115234375, 2.71484375, -0.84619140625, 0.6376953125, 0.11767578125, -0.033447265625, 1.1201171875, 0.03802490234375, 2.712890625, -0.204345703125, -1.0, 1.1513671875, 0.6708984375, -1.59375, 1.2373046875, 0.363525390625, -0.615234375, -0.52685546875, -0.46923828125, 1.79296875, 0.30859375, -1.4931640625, 0.9208984375, -0.96435546875, 0.1370849609375, -0.027679443359375, -0.8076171875, -1.3935546875, 0.94580078125, 0.031829833984375, 0.264892578125, -1.32421875, 1.9033203125, 0.99951171875, 0.583984375, -0.0129241943359375, 0.4560546875, -0.52978515625, -1.3876953125, 0.1727294921875, 0.332275390625, 1.0751953125, 0.362548828125, -0.71240234375, 0.52099609375, -0.36181640625, 0.638671875, 0.132080078125, 0.0210418701171875, 1.17578125, 0.05682373046875, 0.1009521484375, -0.333251953125, 1.7099609375, 1.2529296875, -2.78515625, -1.06640625, 0.73779296875, 0.10675048828125, 0.91552734375, -0.260009765625, 0.8623046875, 0.58544921875, 1.9765625, 1.1328125, -0.52783203125, 0.343994140625, 1.7451171875, -0.07025146484375, -0.479736328125, 1.9169921875, -0.791015625, -0.86669921875, 1.099609375, -0.826171875, 1.5634765625, 0.294189453125, 0.229736328125, -0.10498046875, 0.17236328125, 0.1949462890625, -1.125, 0.061492919921875, 0.91015625, 0.87548828125, -0.2998046875, 0.41748046875, -0.5263671875, 0.3046875, 0.84912109375, 0.396240234375, 0.318603515625, 0.120849609375, -0.09405517578125, 1.005859375, 1.5126953125, -0.6748046875, 0.9677734375, -0.0099334716796875, 0.44677734375, 0.86474609375, 0.50927734375, 1.9951171875, -1.4794921875, -0.59130859375, 1.4619140625, 0.439208984375, -0.29052734375, 0.994140625, 0.2396240234375, 0.6943359375, -0.77734375, -0.775390625, 0.72509765625, -0.38818359375, 0.640625, 1.6259765625, -0.426513671875, 0.183837890625, -1.9091796875, 0.7744140625, -1.5556640625, 1.9345703125, -0.529296875, 0.89404296875, 0.392333984375, 0.08245849609375, 0.281494140625, 2.673828125, 1.853515625, 1.0908203125, -0.6796875, 0.7802734375, 0.93603515625, -0.462646484375, -1.0087890625, -0.03515625, -0.7958984375, 1.7177734375, 0.1566162109375, 0.2061767578125, -1.427734375, 1.1435546875, -0.4619140625, 0.5146484375, -0.2493896484375, 0.64892578125, -0.73681640625, -0.6884765625, -0.0018510818481445312, -1.02734375, 0.3017578125, -0.3974609375, -0.85009765625, -1.01953125, -0.8642578125, 0.92529296875, 1.7060546875, -1.5849609375, -0.0400390625, 0.025726318359375, 0.10198974609375, -0.69287109375, -0.73583984375, 0.184326171875, -0.2286376953125, 1.05859375, -1.345703125, 0.708984375, 2.265625, -0.1514892578125, 0.728515625, -0.642578125, -0.09185791015625, -1.712890625, -1.0771484375, 0.1163330078125, 0.02276611328125, -0.9580078125, -0.3447265625, -0.047393798828125, 1.802734375, -0.7333984375, -1.9072265625, -2.146484375, -0.2197265625, -1.2119140625, -0.03497314453125, -1.4326171875, -1.2490234375, 0.7578125, 0.0399169921875, 0.75, 0.3447265625, 0.434814453125, -0.80810546875, -1.22265625, -0.578125, -0.49267578125, -0.79052734375, 0.71533203125, -0.09686279296875, 0.488037109375, 0.6826171875, 0.4404296875, -1.015625, 0.0234375, -0.015899658203125, 0.005855560302734375, -1.521484375, -0.34521484375, -1.4013671875, -0.55078125, 1.625, 0.75732421875, -1.3291015625, -0.06884765625, 1.423828125, -0.98828125, -2.583984375, 0.52880859375, -0.60107421875, -0.5986328125, -2.0859375, -1.59765625, -0.1275634765625, -0.364013671875, -0.412841796875, 2.337890625, -0.47998046875, 0.0970458984375, 0.39599609375, -0.4599609375, -0.3154296875, 0.268798828125, -1.9375, -1.42578125, 0.496337890625, -0.638671875, -2.00390625, 0.6279296875, 0.0517578125, 1.1533203125, -0.5107421875, -0.9580078125, -1.1962890625, 0.09661865234375, 0.407470703125, 0.64111328125, 0.0229339599609375, 0.4404296875, -0.2286376953125, -0.118408203125, 1.5703125, 1.087890625, -1.1796875, 1.302734375, 0.16259765625, 1.9189453125, -1.0322265625, 0.480712890625, -0.35009765625, 2.1171875, -1.779296875, -0.350830078125, -1.248046875, 0.44580078125, -0.00677490234375, -0.176025390625, -1.234375, 0.4326171875, 1.3115234375, -0.444580078125, 0.69775390625, 1.28125, -0.7587890625, 1.123046875, 0.1912841796875, 0.11541748046875, 1.0517578125, -0.24755859375, -0.364501953125, -1.580078125, 1.79296875, 1.9130859375, 0.77880859375, 0.1123046875, -1.7412109375, 1.169921875, -1.40234375, 0.80712890625, 0.045318603515625, -0.9765625, 0.8310546875, 1.9267578125, 1.1591796875, -1.3935546875, -1.1123046875, -0.45458984375, 1.01171875, 0.7041015625, 0.7333984375, 0.89501953125, -0.94287109375, -0.41845703125, 0.0188751220703125, 0.28125, -0.7236328125, 0.8125, 0.58349609375, -1.0009765625, -1.162109375, 0.99951171875, -0.5126953125, -0.310546875, 2.359375, 0.71728515625, -0.6572265625, 1.4267578125, -1.4521484375, 0.4345703125, -0.11065673828125, -0.0126495361328125, -0.281005859375, 0.92333984375, -0.58740234375, -0.275146484375, 1.376953125, -0.62890625, 0.32275390625, -1.5361328125, -0.82421875, -0.357177734375, 0.7841796875, -0.78076171875, 0.321044921875, 0.912109375, -0.13671875, -1.2900390625, -1.3515625, -0.22021484375, 0.70361328125, -1.294921875, -1.0517578125, -1.1015625, -1.2216796875, -0.265380859375, 1.59375, 0.50146484375, 0.529296875, -0.95947265625, -0.39404296875, 2.19140625, 0.57373046875, 0.1402587890625, -0.744140625, -0.7275390625, -1.1650390625, 0.6611328125, -1.734375, -0.280517578125, 0.83544921875, -1.21875, -1.654296875, 0.74072265625, -0.06591796875, -0.63818359375, -1.2607421875, -2.45703125, 0.255126953125, -1.7607421875, 0.56103515625, 1.1064453125, 0.2337646484375, -0.60400390625, 1.0361328125, -0.32177734375, 0.260009765625, 0.10394287109375, -0.82958984375, -0.24658203125, 1.1044921875, -1.435546875, -0.405517578125, 0.11065673828125, 0.20703125, 1.23046875, 0.71630859375, 0.95263671875, 0.048614501953125, 0.88916015625, -0.302734375, 0.28125, 0.9609375, -0.361328125, -0.438720703125, -0.43115234375, -1.771484375, 0.65380859375, 0.52880859375, -0.40625, 1.2216796875, -0.305908203125, -0.82373046875, 0.263916015625, -0.2156982421875, 0.5791015625, -2.552734375, -1.966796875, 0.95361328125, -0.1583251953125, 1.0458984375, 0.06353759765625, -1.44140625, -1.4873046875, -0.69775390625, 1.2490234375, 2.2265625, 0.3466796875, 0.2354736328125, -1.61328125, -1.5263671875, 0.034423828125, -0.21923828125, -0.111572265625, 1.49609375, -0.75732421875, 0.80859375, 0.1536865234375, 1.978515625, -0.21728515625, -0.083740234375, -2.1484375, 0.2196044921875, -0.179443359375, 1.88671875, -2.0625, 0.7802734375, 2.158203125, 0.638671875, -1.4453125, -0.388671875, -0.86474609375, 1.2099609375, -0.429931640625, 1.1240234375, -0.3701171875, -0.2646484375, 1.3828125, 1.05859375, 0.0689697265625, -0.845703125, 0.12237548828125, 0.859375, -0.434814453125, 1.3935546875, 1.080078125, -0.69189453125, -2.6640625, 0.92041015625, 0.427978515625, 1.7197265625, 1.060546875, 1.1943359375, -0.314208984375, 0.48193359375, 0.97900390625, -1.2265625, -0.9228515625, -0.50439453125, -0.38720703125, 1.7998046875, 0.62060546875, 1.8798828125, 0.68212890625, 0.767578125, -1.21484375, -2.044921875, -1.068359375, 1.1728515625, 1.1962890625, 0.2161865234375, -0.299072265625, -0.78271484375, 0.10986328125, -1.9521484375, 0.62939453125, 1.732421875, 1.220703125, 0.619140625, -0.95556640625, -0.82763671875, -0.31640625, -0.9169921875, -0.041046142578125, -0.75, -0.20361328125, -0.35302734375, -0.143310546875, -1.5048828125, 1.4833984375, 0.71142578125, 1.51953125, -1.0947265625, 0.2037353515625, 1.2998046875, 0.29833984375, 2.154296875, -0.69970703125, -0.06439208984375, -0.2098388671875, -0.58447265625, -0.467529296875, -1.126953125, -0.52685546875, -0.016204833984375, 1.0068359375, 2.330078125, -0.59375, -0.458251953125, 0.25390625, 0.92626953125, 0.6875, -0.197265625, 1.8212890625, -0.07794189453125, -0.446533203125, 1.0380859375, 1.005859375, 0.1995849609375, 1.3603515625, -0.2430419921875, -0.65478515625, -0.371337890625, -0.447509765625, 1.4150390625, 1.9853515625, 1.234375, -0.83642578125, 0.367919921875, 1.8759765625, 0.22119140625, 0.01071929931640625, -0.006412506103515625, 0.8564453125, -2.296875, -0.61669921875, -0.58056640625, 0.9287109375, -1.826171875, 0.1766357421875, 1.4375, -0.2459716796875, 2.513671875, -0.9716796875, 1.7978515625, -1.064453125, 0.51025390625, 2.2421875, 0.681640625, 0.76123046875, 0.2371826171875, 0.38232421875, -0.8466796875, -0.342041015625, -1.603515625, 0.28857421875, -0.8330078125, 1.373046875, 0.80712890625, 0.904296875, 1.7392578125, -0.5498046875, 0.2342529296875, 0.52978515625, 0.406982421875, 2.556640625, -0.54931640625, 1.4091796875, -0.284423828125, 1.294921875, 0.6220703125, -0.50244140625, -1.6123046875, -0.91748046875, 1.353515625, 0.6513671875, -0.15283203125, -0.63525390625, 0.66650390625, 1.0322265625, 1.1181640625, -1.068359375, 0.61083984375, 0.74951171875, -0.66845703125, -0.3232421875, 0.89697265625, 0.1341552734375, -1.33203125, -0.425537109375, -1.44921875, -0.0970458984375, -0.332763671875, -0.0012645721435546875, 2.1328125, 1.5322265625, 1.236328125, 0.305419921875, -0.271484375, 0.4052734375, 1.560546875, 1.314453125, -1.1572265625, -0.479736328125, 1.2822265625, -1.15234375, -0.69970703125, 0.6396484375, 0.60888671875, -0.98828125, -1.1689453125, -0.04388427734375, -0.04632568359375, -0.87939453125, -0.05487060546875, 0.070068359375, -1.4248046875, 1.302734375, 0.388427734375, -1.1552734375, -0.52392578125, 0.23193359375, 0.84423828125, 0.0631103515625, 0.69287109375, -1.3701171875, 0.740234375, 0.8603515625, 0.58935546875, 0.09576416015625, 0.86328125, -0.390869140625, -0.43603515625, 0.2587890625, -0.330322265625, -0.1036376953125, 0.3671875, -1.57421875, -0.482666015625, -0.008148193359375, 0.49462890625, 0.6015625, 0.45703125, -1.1044921875, 1.8349609375, -1.115234375, -0.3701171875, -1.421875, 0.53076171875, 0.25537109375, -0.59814453125, 0.52392578125, 0.34765625, -0.089111328125, 1.845703125, 1.2841796875, 2.044921875, -1.0888671875, -1.478515625, 1.4228515625, 1.134765625, 0.54736328125, 1.2939453125, -0.006984710693359375, 1.6240234375, -0.69189453125, 1.5, 1.068359375, -0.94970703125, 0.5126953125, -1.015625, -0.371826171875, 1.1318359375, 1.5517578125, -0.8935546875, 0.65380859375, -1.384765625, -0.6875, -0.1751708984375, -1.1796875, -0.6943359375, 0.0242156982421875, 2.89453125, -0.7158203125, -0.0369873046875, -0.69140625, -1.642578125, 1.16796875, 0.7646484375, -1.732421875, -0.5546875, -0.271484375, 0.62451171875, 0.27880859375, 0.36279296875, -0.258056640625, -0.57177734375, -0.49560546875, 0.190185546875, 0.4736328125, -0.0826416015625, 0.94140625, -0.059783935546875, 0.353271484375, 0.2353515625, -0.47265625, -0.48095703125, 0.54541015625, 1.359375, -1.2412109375, -0.96044921875, 0.240478515625, -1.1962890625, -0.6796875, 0.489013671875, 0.724609375, 0.389892578125, -1.1279296875, 0.6494140625, 0.79736328125, 0.379638671875, 1.5712890625, 0.5771484375, 0.476806640625, 1.1337890625, -0.55908203125, -2.521484375, 0.438720703125, -1.5498046875, 0.3203125, 0.473388671875, -0.62841796875, 1.447265625, 0.13525390625, 0.51904296875, -0.61328125, 1.69921875, 2.580078125, 0.445068359375, -0.7275390625, -0.021148681640625, 0.309326171875, -0.7939453125, 0.58740234375, 0.1561279296875, 0.153564453125, -0.2459716796875, 0.049224853515625, 0.43115234375, -1.783203125, -1.7568359375, -2.203125, 0.103515625, -0.1925048828125, 1.9130859375, -0.17724609375, 1.6806640625, -0.47021484375, 0.03253173828125, -0.1304931640625, -1.64453125, -0.12548828125, -1.15234375, 0.28271484375, -0.328125, -0.2093505859375, -0.419189453125, -1.439453125, -0.85693359375, -0.90625, -1.8466796875, 0.165283203125, -1.1962890625, -0.984375, 1.0546875, 0.64404296875, -0.9794921875, -0.755859375, -0.386962890625, -0.268310546875, -1.55859375, 1.087890625, 0.97265625, -0.80712890625, 0.04461669921875, -1.3994140625, 0.79736328125, -0.173095703125, 0.1737060546875, 0.49169921875, 0.94384765625, -1.736328125, 0.59814453125, 0.7861328125, -0.638671875, 0.0236053466796875, 1.556640625, 0.61279296875, -0.52001953125, -1.1884765625, 0.98193359375, -0.52978515625, 0.94091796875, 1.1298828125, -0.818359375, 0.145263671875, 0.237548828125, 0.031982421875, -0.353271484375, -0.86474609375, 0.763671875, -0.94580078125, 0.40185546875, -0.170654296875, 0.2388916015625, 0.4462890625, -0.143798828125, 0.94580078125, -0.34130859375, 0.21875, -0.2357177734375, 0.52978515625, -0.83203125, -0.061767578125, -1.2958984375, 0.302978515625, 0.173828125, 1.048828125, -0.19970703125, -0.007511138916015625, -0.2174072265625, 0.64501953125, 0.0021076202392578125, -0.08111572265625, 1.5234375, -0.37890625, 0.17138671875, 0.0860595703125, 1.1494140625, -1.0869140625, 1.2314453125, 0.6259765625, -1.2578125, -0.89453125, -0.1766357421875, -0.64697265625, 0.56884765625, 2.505859375, 0.74072265625, 1.1640625, 1.0029296875, 0.90625, -1.314453125, 1.201171875, -1.197265625, -0.1522216796875, 1.0166015625, -0.6689453125, -0.736328125, 0.09820556640625, 0.1533203125, 0.00780487060546875, -0.236572265625, -0.0374755859375, -0.22705078125, 0.017486572265625, 2.046875, 0.408447265625, -0.92041015625, -1.19921875, 0.685546875, 0.63427734375, 0.330322265625, -0.7333984375, -1.376953125, 1.11328125, 1.0625, 0.9775390625, 0.48095703125, -0.343505859375, 0.30908203125, 1.4677734375, -0.5634765625, 0.43603515625, 0.697265625, 1.7197265625, 1.984375, 0.8857421875, -0.93017578125, -0.10906982421875, 0.7353515625, -0.270751953125, -0.7763671875, 0.309326171875, 0.810546875, 0.89599609375, 0.30322265625, 0.243896484375, -0.5712890625, 0.89599609375, -1.728515625, 0.10260009765625, 0.90234375, -1.0732421875, 1.6083984375, 0.841796875, -1.265625, 0.07489013671875, 1.658203125, -0.386474609375, -1.16015625, 1.4296875, -0.63232421875, 0.74853515625, 0.75390625, -1.1650390625, -0.00525665283203125, 0.6220703125, -0.1201171875, 0.45166015625, -0.6083984375, -0.7890625, -0.380126953125, -0.94970703125, -0.9521484375, -1.0693359375, 0.381591796875, -1.619140625, 0.4775390625, -0.456787109375, -0.353759765625, -0.7734375, -0.313720703125, -1.171875, 1.115234375, -0.472900390625, 1.1220703125, 2.466796875, -0.448486328125, -1.818359375, 1.197265625, 0.1463623046875, 0.72998046875, -0.037750244140625, -0.403076171875, -0.82568359375, 0.2130126953125, -0.73974609375, -0.2841796875, 1.45703125, 0.52587890625, -0.73876953125, -0.285888671875, -0.036895751953125, 0.295654296875, 0.247314453125, 0.184326171875, -1.3193359375, 2.265625, -0.39599609375, 0.344970703125, 2.70703125, -2.525390625, -0.20654296875, 0.48681640625, 0.36865234375, 1.1884765625, 0.33544921875, -1.33984375, -1.12109375, -0.66796875, -1.337890625, -0.67578125, 1.15625, -0.23779296875, -0.8310546875, 0.70166015625, 0.32373046875, 0.36767578125, 0.0928955078125, 1.96484375, -0.407958984375, -0.5634765625, -1.0234375, -1.287109375, -0.697265625, 0.857421875, -0.255126953125, 0.68505859375, -0.43798828125, 1.26953125, -1.4912109375, 0.2081298828125, 0.46142578125, 0.366943359375, 0.75634765625, -0.197021484375, -0.912109375, 1.1875, 0.346923828125, 1.21484375, 0.05816650390625, -0.15234375, 0.6396484375, -1.6318359375, -1.02734375, 0.36083984375, 0.40380859375, -1.4423828125, 0.9306640625, -0.50244140625, 2.0546875, 0.6181640625, 0.2342529296875, 1.6455078125, -0.238525390625, -0.89404296875, -0.8828125, -0.81591796875, -0.3662109375, -0.701171875, -1.1591796875, -0.303955078125, -0.435791015625, 0.5732421875, 0.92529296875, 0.1278076171875, -0.60986328125, -0.363037109375, 0.4384765625, 0.72998046875, 2.130859375, 0.04376220703125, -0.60400390625, 0.7421875, 0.426513671875, 0.9775390625, 2.060546875, 1.1220703125, 1.0908203125, 1.1328125, 1.314453125, -2.40234375, 0.32470703125, -1.9970703125, 0.66162109375, 0.386474609375, 0.8291015625, -0.455810546875, 0.333740234375, -0.0330810546875, -0.490478515625, 0.72900390625, -1.1630859375, 0.51953125, -0.978515625, -0.63232421875, -0.79638671875, 0.09912109375, -1.431640625, -1.822265625, 1.6708984375, 0.494384765625, -0.15673828125, -1.5078125, 0.087646484375, -0.488525390625, -1.04296875, 0.90087890625, -1.3193359375, -0.01324462890625, 1.0478515625, -0.1436767578125, -1.376953125, -0.7373046875, -1.845703125, 1.0380859375, 0.2393798828125, 1.5732421875, 1.353515625, -0.439453125, -0.9775390625, 0.66357421875, -0.21533203125, -1.8818359375, 1.154296875, 1.70703125, -0.75048828125, -0.354248046875, 1.5986328125, -0.64794921875, 0.5078125, -0.68115234375, -1.0556640625, 1.0595703125, -0.10040283203125, -0.736328125, -0.74951171875, -0.5302734375, 0.32421875, 0.79638671875, -0.71044921875, -1.8017578125, 0.8701171875, -0.8515625, 0.517578125, -2.099609375, -0.350830078125, 0.02716064453125, -0.328857421875, -1.1455078125, -1.408203125, 0.07275390625, -0.411865234375, 0.947265625, -0.320556640625, -0.5732421875, 0.61962890625, 0.3857421875, 1.173828125, 0.325439453125, 0.457275390625, -1.2197265625, -2.826171875, -1.6884765625, -0.27783203125, -1.3232421875, 0.44921875, 0.031707763671875, -1.30859375, 0.1497802734375, -0.47265625, -0.0242919921875, -0.9716796875, 0.775390625, -0.779296875, 0.517578125, 1.84765625, -0.20263671875, -2.328125, -1.236328125, -0.185546875, 1.16796875, -2.900390625, 0.70751953125, -0.35400390625, -0.2998046875, 2.1484375, -2.056640625, -0.313232421875, 0.611328125, 0.53515625, -0.8330078125, -1.9775390625, -0.462158203125, -0.471923828125, -0.33349609375, -0.48291015625, -1.7412109375, -1.0927734375, -0.2646484375, -1.0576171875, 0.49853515625, 1.677734375, 0.609375, -1.7431640625, 1.0888671875, -2.45703125, 1.583984375, -0.1910400390625, 0.031158447265625, -0.44970703125, -0.006786346435546875, -2.037109375, -0.465087890625, -3.15234375, -0.11248779296875, 0.650390625, -0.04046630859375, 0.89111328125, -0.495361328125, 0.43896484375, 0.54931640625, 0.0285186767578125, 0.04901123046875, 0.509765625, 0.0207366943359375, -0.11297607421875, -0.1259765625, 0.049224853515625, 0.7373046875, -2.111328125, 0.97705078125, -1.375, -1.4423828125, -0.46435546875, 0.200927734375, 0.82861328125, 1.37890625, 0.70166015625, 0.262451171875, 0.88232421875, 1.3759765625, 0.39990234375, -1.939453125, 0.87890625, 1.865234375, 0.556640625, -0.50439453125, 0.032501220703125, 1.046875, -1.48828125, -0.361083984375, -0.1502685546875, -1.1728515625, 1.2958984375, 2.310546875, 0.108154296875, -2.068359375, -0.1650390625, -1.255859375, -0.2271728515625, -0.1883544921875, -1.4951171875, -0.54443359375, -0.294189453125, 0.292724609375, -0.87646484375, 0.07403564453125, 1.052734375, 0.5244140625, 1.1171875, -0.77197265625, -0.376953125, -0.314208984375, -0.625, -0.61083984375, -1.0400390625, 0.66748046875, -0.36572265625, 0.09283447265625, 1.1904296875, -0.55615234375, 0.99658203125, 2.06640625, 0.73974609375, 0.1864013671875, -0.85107421875, 0.4775390625, 1.3154296875, 0.5302734375, 1.357421875, 0.359375, -0.53955078125, -0.90869140625, -0.384033203125, 0.427734375, -0.5498046875, 0.42578125, 0.2015380859375, -0.2176513671875, -0.60107421875, 0.1982421875, -0.3291015625, -0.67822265625, -1.48046875, -0.2095947265625, 1.146484375, 0.00618743896484375, 0.26708984375, -0.33740234375, -0.79052734375, 0.288330078125, -0.378173828125, -0.79443359375, -0.57275390625, 0.262451171875, 0.278564453125, 0.67236328125, 0.3125, 1.78125, -0.034759521484375, 0.6630859375, 1.49609375, -0.302734375, -0.036590576171875, -0.065673828125, -0.072998046875, 0.57275390625, 1.857421875, -0.130859375, 0.69970703125, -0.26416015625, 1.9228515625, -0.70849609375, 1.0517578125, 1.166015625, -1.171875, 0.72607421875, -0.042755126953125, -0.2490234375, 0.05975341796875, -0.0298919677734375, 1.193359375, 2.470703125, -0.6708984375, 1.6201171875, -0.389404296875, 0.951171875, -0.57861328125, -2.2734375, -0.07049560546875, 0.845703125, 0.313720703125, 0.474853515625, 1.0859375, 0.85107421875, -2.423828125, 0.32666015625, -0.58984375, 1.5341796875, 0.306640625, 0.677734375, -0.68798828125, -1.234375, 0.736328125, -0.262451171875, -2.01171875, 0.802734375, -1.83203125, -1.556640625, -0.480224609375, -0.84521484375, -0.923828125, -1.501953125, 0.428955078125, 0.9697265625, 1.5107421875, -0.49169921875, 1.0087890625, -0.39306640625, 1.6455078125, -1.15234375, -0.054168701171875, 1.525390625, -1.30859375, 0.151611328125, 0.395751953125, 1.4814453125, 0.342041015625, 2.087890625, 1.5517578125, -0.1649169921875, 0.10888671875, -0.7841796875, -0.385986328125, -0.640625, -0.82958984375, 0.53466796875, -0.94189453125, -0.041595458984375, 0.383056640625, 0.294921875, -2.21875, 1.3720703125, 1.0185546875, 1.376953125, -0.0240478515625, 0.0775146484375, 0.037506103515625, 1.7607421875, 2.501953125, 0.974609375, -1.130859375, 1.697265625, 1.3896484375, -0.30078125, 0.677734375, -2.3359375, 2.34765625, -1.8447265625, -0.1546630859375, 0.032684326171875, -1.10546875, -0.345947265625, -0.0284881591796875, 0.468017578125, 0.22509765625, -0.468994140625, 0.122802734375, -0.09906005859375, 0.84912109375, 1.68359375, -0.64599609375, 0.0960693359375, 1.357421875, -0.319580078125, -0.06561279296875, -0.309814453125, 0.1046142578125, 0.71044921875, -1.150390625, -0.03155517578125, 1.271484375, 0.80810546875, 1.0927734375, -1.0048828125, -1.9814453125, 0.364990234375, 0.375244140625, 1.9052734375, -0.234619140625, -0.51513671875, -0.5859375, -0.89306640625, -0.84814453125, 1.2744140625, 1.859375, 0.7685546875, -1.6318359375, 1.8369140625, -1.03125, -0.5322265625, 0.24072265625, 1.4052734375, -0.8115234375, 1.2255859375, 0.134033203125, 0.79345703125, -0.77001953125, 0.2352294921875, -0.331298828125, -0.95361328125, -0.56494140625, -0.07318115234375, -0.408447265625, 0.681640625, -1.56640625, 0.1854248046875, -0.3408203125, -0.97119140625, -3.49609375, 0.1556396484375, -0.513671875, 1.66796875, 0.032073974609375, -1.359375, 0.442138671875, -0.421630859375, 1.1396484375, -0.75439453125, 0.5439453125, -1.724609375, -0.0198822021484375, -0.75537109375, 0.1890869140625, 0.50439453125, -0.2303466796875, 0.6767578125, -1.1162109375, -0.98681640625, -0.90625, -0.379150390625, 0.2021484375, 1.1533203125, -0.400634765625, 0.5419921875, -0.12384033203125, 0.08795166015625, 1.1259765625, 0.484375, -0.7724609375, -0.30224609375, 1.3291015625, -0.44775390625, -1.2099609375, -0.673828125, 1.8681640625, -0.67431640625, 0.398193359375, 0.10345458984375, -1.5078125, 0.018707275390625, 2.73828125, -0.99072265625, 0.6220703125, 0.463134765625, -1.759765625, -0.3564453125, -0.42236328125, 0.3505859375, -0.42626953125, 0.56884765625, 0.9453125, -0.31396484375, 2.556640625, 1.25, -1.359375, -0.6767578125, 0.53857421875, -0.1771240234375, -0.1453857421875, 0.171630859375, 0.05450439453125, 2.0390625, 0.68994140625, 0.5693359375, 0.73583984375, -0.78857421875, -0.491455078125, -0.3115234375, -0.27783203125, 0.331787109375, 0.424072265625, -0.509765625, -0.84033203125, -0.11297607421875, -1.1044921875, 0.04534912109375, 0.3193359375, 1.0078125, -0.05224609375, 0.5263671875, -0.62255859375, -0.410400390625, -0.005596160888671875, 0.77783203125, -0.289794921875, -0.80322265625, 0.4853515625, -1.724609375, 1.611328125, -4.078125, -1.560546875, 0.1619873046875, 0.3203125, 0.004451751708984375, -0.169189453125, -1.2470703125, 0.484130859375, 0.4814453125, -0.7294921875, -0.2578125, -0.68017578125, 0.1632080078125, -1.0654296875, -0.131103515625, -0.184814453125, -0.1881103515625, 0.57470703125, -0.0041656494140625, -1.76953125, -1.7578125, 0.85595703125, -0.92626953125, -1.990234375, 0.207763671875, 1.2177734375, -1.005859375, 0.2322998046875, 1.1533203125, 0.435791015625, 0.390380859375, -0.408447265625, 1.19921875, -0.6376953125, -0.281005859375, 0.84033203125, -0.92822265625, -0.84521484375, -0.08319091796875, -1.634765625, 0.1927490234375, -0.82421875, -0.52001953125, 0.103271484375, 1.1064453125, -1.9921875, 0.84423828125, -1.162109375, 0.4150390625, -0.1429443359375, -2.3828125, -0.0462646484375, 0.0675048828125, 2.59375, 1.1640625, -1.29296875, -0.1268310546875, 1.0, 1.3857421875, 0.77734375, -0.48388671875, -1.5, -0.12469482421875, 0.47705078125, 1.0693359375, -1.0986328125, 1.2138671875, -0.1370849609375, -0.473876953125, 1.4033203125, 1.7509765625, -0.74755859375, 0.321044921875, 0.2108154296875, 1.509765625, 0.5224609375, 0.9814453125, -0.67724609375, 0.77783203125, 1.330078125, 0.195556640625, 0.242431640625, -1.451171875, -1.7099609375, -2.037109375, -1.853515625, -0.2403564453125, -1.904296875, -0.06451416015625, -0.256591796875, 0.646484375, -0.8994140625, -0.0999755859375, 0.01297760009765625, 0.283935546875, 0.552734375, 0.46435546875, 1.32421875, -2.3359375, 0.164306640625, -1.7568359375, -0.658203125, 2.93359375, -0.9892578125, -0.3232421875, 0.007518768310546875, 0.75341796875, -0.3994140625, -1.5771484375, 2.244140625, 0.2254638671875, 0.452392578125, -0.2666015625, 1.05859375, 0.68017578125, 1.1279296875, 1.4580078125, 1.04296875, 1.169921875, -0.08074951171875, 1.3046875, -0.394287109375, 1.1015625, -0.6904296875, -0.43212890625, -0.378173828125, -0.79638671875, -0.85107421875, 0.273681640625, -1.20703125, 0.263916015625, 0.65966796875, -0.79248046875, 0.52734375, -0.138427734375, 1.5703125, 0.0794677734375, 1.6220703125, -0.2783203125, -0.402099609375, -0.875, -0.6435546875, 0.159423828125, -0.281982421875, -0.6865234375, -1.28125, -0.1151123046875, 1.59375, 0.4345703125, 0.1593017578125, 0.46240234375, -0.5810546875, -1.1865234375, -1.3125, -0.677734375, -0.2177734375, 0.5654296875, -0.509765625, -1.6513671875, -0.1600341796875, 1.3525390625, 0.0782470703125, 1.375, -0.66943359375, 0.0213775634765625, -1.1435546875, 0.278564453125, -0.023345947265625, 1.4873046875, 0.337890625, -0.92041015625, 1.46875, 1.4931640625, -0.0264739990234375, -0.11865234375, 0.83251953125, -0.732421875, 2.4140625, 0.23291015625, 0.33056640625, 0.36669921875, -0.0193939208984375, 0.2286376953125, -0.54833984375, 0.13720703125, -0.1124267578125, -0.97314453125, 0.17236328125, -0.8017578125, -1.5205078125, 1.0634765625, -1.455078125, 2.16796875, 0.8486328125, 0.634765625, 0.35595703125, -1.7578125, -0.54833984375, 0.68798828125, 0.6552734375, 0.400390625, -0.50390625, 0.53759765625, -0.34375, -0.341796875, 2.177734375, 0.412353515625, 0.9697265625, 0.267822265625, -0.3642578125, 0.1622314453125, 0.467041015625, -0.31884765625, -0.9794921875, -0.947265625, -0.255859375, -1.0498046875, -0.66845703125, -0.7939453125, -0.2529296875, -0.413818359375, 0.6337890625, -0.62744140625, -0.387939453125, -1.19921875, 0.1405029296875, -0.791015625, -0.6044921875, -0.287353515625, -0.8388671875, -0.63916015625, -0.0038509368896484375, -1.7978515625, 0.8408203125, -0.059478759765625, 1.7998046875, -0.9814453125, 0.3369140625, 0.049957275390625, 1.724609375, 0.87451171875, 1.21875, -0.425048828125, 2.390625, -0.02923583984375, -1.4296875, 0.354736328125, -0.56201171875, 0.27294921875, 0.2496337890625, 1.671875, -0.326171875, -0.368408203125, -1.8447265625, 0.72119140625, 0.054901123046875, -0.8388671875, -0.7548828125, -0.0225067138671875, -3.91015625, 2.46484375, -1.326171875, 1.05859375, 0.358642578125, -0.0010528564453125, -0.46484375, -0.62353515625, -0.88134765625, 1.232421875, -0.2467041015625, 0.6513671875, -0.0364990234375, 0.01421356201171875, -1.0263671875, 0.1383056640625, 1.193359375, 0.76220703125, 1.1591796875, 1.4951171875, -0.916015625, 0.78466796875, 0.57666015625, -2.240234375, 0.56201171875, 1.490234375, 0.320556640625, 0.1085205078125, -1.4150390625, 0.671875, -1.646484375, 0.334716796875, -0.98095703125, 1.779296875, 0.6259765625, -1.0576171875, -0.97314453125, -1.529296875, -0.3125, 0.6796875, 1.2783203125, 0.8837890625, -0.11285400390625, 1.2939453125, -1.47265625, 1.3212890625, -0.09503173828125, -0.1546630859375, 0.06719970703125, 0.083740234375, -1.3935546875, -0.84521484375, 0.6796875, 1.0498046875, 0.4814453125, -0.27001953125, -2.07421875, -0.1217041015625, -0.1990966796875, 0.7255859375, -1.74609375, 0.76025390625, 0.2186279296875, 1.2744140625, 0.07025146484375, 0.90771484375, -1.560546875, 1.8642578125, -0.4443359375, 0.716796875, -0.470703125, -0.15185546875, -0.1419677734375, 1.3671875, -1.2138671875, -0.149658203125, 1.7783203125, 1.205078125, -0.1534423828125, 0.10455322265625, -0.826171875, -0.3359375, -0.759765625, -0.078857421875, 0.09039306640625, -0.60791015625, 0.1287841796875, -0.286376953125, 2.333984375, -0.59619140625, 1.025390625, -1.71484375, -1.2197265625, 0.775390625, -1.4775390625, -1.0888671875, 0.401611328125, 0.256591796875, -0.422119140625, 0.059173583984375, -1.546875, 0.81591796875, 0.44287109375, 0.37353515625, 1.0537109375, 0.172119140625, 1.205078125, 2.279296875, 0.96337890625, -1.8916015625, 0.28125, -0.00719451904296875, -0.52783203125, -1.6513671875, 0.98779296875, -1.654296875, 2.34765625, -0.2939453125, -0.347900390625, 1.14453125, 0.958984375, -1.291015625, -0.52587890625, 0.1806640625, 0.7275390625, 0.7939453125, 1.80859375, -1.1533203125, -0.1497802734375, -0.72021484375, 0.8095703125, -0.7275390625, -1.921875, 0.7490234375, 0.134765625, -1.0654296875, -1.154296875, -1.4658203125, -0.345703125, -0.37060546875, -1.4208984375, 1.0244140625, 1.15625, -0.5654296875, -0.376953125, -0.2464599609375, 1.1103515625, 1.392578125, -0.038482666015625, 0.79296875, 0.44482421875, 1.2353515625, -0.489501953125, -1.71875, -0.35791015625, 2.8125, -1.1015625, 0.11126708984375, 1.94140625, 0.1781005859375, -0.07391357421875, -0.47412109375, -0.3603515625, 0.1593017578125, 0.8408203125, 0.5654296875, -0.64404296875, -0.1402587890625, -0.60546875, -2.794921875, -2.05078125, -1.8583984375, 0.7900390625, -0.2467041015625, 1.4189453125, 1.0166015625, 2.27734375, 0.73681640625, -0.80078125, 1.203125, -0.99609375, -0.6923828125, 1.05859375, 0.251953125, -1.0625, 1.033203125, 0.98876953125, 0.56396484375, 0.68359375, -0.9462890625, 0.045989990234375, -0.390380859375, 1.26953125, 0.43359375, 0.26025390625, 0.06854248046875, 0.39892578125, 0.51953125, 1.859375, 0.67724609375, -0.031585693359375, 1.7587890625, 1.5693359375, -0.323974609375, -0.489013671875, 0.09185791015625, 0.0214385986328125, -0.2724609375, -0.3837890625, -0.449951171875, -1.1298828125, 0.9052734375, 1.6767578125, 0.474853515625, -0.278564453125, -1.970703125, 1.6044921875, -0.491943359375, 0.7392578125, 0.5458984375, 0.6826171875, 1.8447265625, -1.1201171875, 1.0, -2.21484375, -0.67919921875, -1.2734375, 0.2041015625, 0.217041015625, -1.6474609375, -0.263427734375, 1.626953125, -1.126953125, -0.1717529296875, 1.525390625, -2.37109375, 0.2322998046875, 0.96044921875, -0.66357421875, -0.17333984375, -1.3212890625, 0.1702880859375, -0.385498046875, 2.25, 1.6806640625, 0.340087890625, -0.76953125, -0.2408447265625, 0.4208984375, -0.01238250732421875, 0.273193359375, -0.32080078125, 1.2001953125, -0.2408447265625, 0.055511474609375, 0.378173828125, 1.0908203125, 0.194580078125, 0.6728515625, -1.330078125, -0.62109375, 0.673828125, -0.72021484375, -0.1500244140625, 0.49072265625, -1.0927734375, -0.309814453125, -0.87158203125, -0.25390625, 1.6201171875, -0.32275390625, -0.042327880859375, -0.481201171875, -0.420654296875, -0.43798828125, -2.1328125, -0.6611328125, -0.876953125, -0.92431640625, -1.2958984375, 1.8583984375, -0.0186614990234375, -0.1834716796875, -1.7939453125, 2.98828125, 0.5654296875, -1.0654296875, -0.6064453125, -0.6181640625, -0.39599609375, -0.488525390625, 0.022735595703125, -1.1669921875, -0.312255859375, 1.119140625, 0.458984375, -0.21826171875, 1.4228515625, 0.2454833984375, 0.552734375, -0.037109375, 0.87548828125, 1.357421875, 1.3349609375, -0.79248046875, -0.09552001953125, -1.302734375, 0.320068359375, 0.1849365234375, -1.2412109375, 0.70166015625, -0.51806640625, 0.83056640625, 1.09765625, -0.133056640625, 0.9599609375, -1.69140625, 1.4189453125, -1.16015625, -0.38525390625, 0.00751495361328125, -1.0107421875, -0.461181640625, -0.350341796875, -0.266357421875, 0.27392578125, -0.37353515625, 0.1490478515625, -0.0011072158813476562, -0.140625, 0.75927734375, -1.73046875, -0.30908203125, 0.50830078125, -0.08447265625, 0.3173828125, -0.2578125, -0.494873046875, 0.208984375, 1.12890625, -0.61181640625, 1.71484375, 1.37109375, -0.85546875, 0.5693359375, -0.20068359375, 0.150146484375, 0.1954345703125, -0.398681640625, 1.201171875, 0.9697265625, -1.1142578125, -0.98876953125, -1.1220703125, 0.92041015625, -0.513671875, -0.69921875, -0.60107421875, -0.9677734375, -1.55078125, -1.0009765625, 0.27685546875, -2.029296875, -0.196044921875, 0.434814453125, 0.2822265625, 0.5966796875, -0.2469482421875, 0.51708984375, 1.345703125, 2.025390625, -0.72900390625, -0.08489990234375, 0.6669921875, 0.442626953125, 0.494873046875, 0.92626953125, 0.038848876953125, -0.6806640625, 0.68798828125, 0.09912109375, -0.5869140625, 1.8564453125, 1.0498046875, 0.2479248046875, -0.488525390625, 0.59326171875, 0.939453125, -0.346435546875, -0.122802734375, -0.406005859375, 1.7353515625, 2.3671875, -0.396728515625, 0.308349609375, -0.01352691650390625, -0.271728515625, 0.2685546875, -0.410888671875, 0.332763671875, 0.2271728515625, -0.998046875, 0.990234375, 0.495361328125, 1.916015625, 0.97998046875, -0.6103515625, 1.177734375, 0.203369140625, -0.96484375, -2.537109375, -0.059814453125, -0.68212890625, 1.2060546875, -0.2149658203125, -0.129150390625, 0.404296875, -1.576171875, 0.49951171875, -0.036376953125, 1.6474609375, -1.7265625, 0.8310546875, -1.5849609375, -0.224365234375, -0.18798828125, 1.8408203125, 0.277587890625, -0.423095703125, 0.37744140625, -0.50732421875, -0.290283203125, -0.9990234375, -0.0618896484375, 1.2802734375, 0.188720703125, 0.04742431640625, 1.2470703125, -0.12255859375, -0.26025390625, -0.5458984375, 1.89453125, -0.73046875, -3.384765625, 0.493408203125, -2.123046875, -0.66796875, -0.052947998046875, -1.208984375, -1.4697265625, 1.4306640625, -2.294921875, -0.8388671875, 0.28564453125, 1.982421875, 0.09771728515625, 1.1806640625, -1.203125, -1.2197265625, -0.11328125, -0.2900390625, 0.2342529296875, -1.7978515625, 0.9013671875, 0.237060546875, 0.17919921875, 0.2939453125, 0.390380859375, 0.3291015625, 1.3076171875, -1.482421875, -0.7958984375, 1.677734375, -0.3291015625, -0.93017578125, 2.28125, 1.3876953125, -0.0733642578125, 0.57470703125, 0.320556640625, -0.56005859375, -0.7158203125, 2.05078125, 0.77685546875, -0.775390625, -0.91259765625, -0.408447265625, 0.814453125, 0.76904296875, -0.82861328125, -0.638671875, -0.9140625, 0.457763671875, 2.064453125, 0.41162109375, -0.146240234375, 0.5146484375, 0.96337890625, -0.3125, -1.1943359375, 1.564453125, 0.027191162109375, -0.400634765625, 0.357421875, 1.2724609375, -1.373046875, -0.98779296875, 0.95068359375, 0.384033203125, -0.197265625, -0.323974609375, 0.5361328125, -0.88232421875, 1.2099609375, -0.495361328125, -0.52197265625, -1.65625, -1.7626953125, -2.638671875, -2.62890625, 0.12359619140625, 0.0239105224609375, -0.8251953125, -0.29541015625, -1.32421875, -0.093994140625, -0.2322998046875, -0.87158203125, -2.31640625, -0.2225341796875, -0.40625, -0.262939453125, -1.984375, 0.1253662109375, -0.09307861328125, -1.626953125, 0.66064453125, 0.77978515625, -1.728515625, 0.2607421875, 0.00991058349609375, -2.71484375, -1.6826171875, -0.3115234375, 0.93115234375, -1.849609375, -0.56494140625, -0.259521484375, 0.427734375, 1.6572265625, -0.20361328125, 0.81591796875, -1.193359375, 0.51025390625, 1.9833984375, -1.1845703125, 0.955078125, 0.4326171875, -0.067626953125, 0.2802734375, 1.3798828125, 0.552734375, 1.2509765625, 1.5185546875, -0.873046875, 0.74951171875, -0.62744140625, -0.041015625, 0.1605224609375, 0.198486328125, -0.479248046875, 0.79931640625, 0.89794921875, 0.59814453125, -0.2822265625, 0.06329345703125, -0.68408203125, -0.91064453125, 1.2099609375, 0.2236328125, 0.2841796875, 0.34716796875, 0.83251953125, -0.5185546875, 0.51123046875, 0.873046875, -0.206298828125, 1.298828125, -0.4970703125, 0.66259765625, 1.494140625, -0.258056640625, -1.13671875, 1.2314453125, -0.5888671875, -0.4677734375, -0.355224609375, 0.76220703125, -0.78564453125, 0.55517578125, 0.8564453125, 0.400146484375, -0.28271484375, 0.46044921875, -0.397705078125, -0.1978759765625, 0.186767578125, 0.6435546875, 0.05145263671875, 0.7275390625, 0.041351318359375, 1.4228515625, 0.9814453125, 0.8955078125, -1.1826171875, 0.108642578125, -1.2099609375, 0.2271728515625, -0.67041015625, -0.337890625, -0.2386474609375, -0.52685546875, -1.2783203125, 1.046875, -0.6337890625, -0.3330078125, 0.2078857421875, -0.47412109375, 0.409912109375, 0.83935546875, 0.2215576171875, -1.1162109375, -0.6767578125, 1.2099609375, -0.267578125, -1.3974609375, -1.6171875, 0.251953125, -0.038970947265625, 0.8115234375, 0.9921875, -2.474609375, -0.386474609375, 0.48193359375, -0.99658203125, -1.466796875, -0.14990234375, -0.75390625, 0.1922607421875, 0.685546875, 1.8720703125, 1.857421875, -0.34375, -1.1162109375, 1.5146484375, -1.02734375, 0.5810546875, -2.240234375, 2.46875, 0.331787109375, 1.7705078125, 2.212890625, 0.6875, -1.572265625, 0.0296478271484375, 0.73388671875, 1.41015625, 0.370361328125, 1.130859375, 0.425048828125, 1.21875, 2.029296875, 1.1484375, -0.26318359375, 1.837890625, -0.6376953125, -1.294921875, -1.3671875, -0.104248046875, -0.60302734375, 1.2958984375, 1.7587890625, -0.77392578125, -1.484375, 1.7060546875, 1.1845703125, -0.08734130859375, 0.2376708984375, 1.4375, -1.740234375, -0.84326171875, 2.369140625, 1.7724609375, 0.859375, -1.2900390625, 0.371826171875, 0.3662109375, -0.81982421875, -1.3984375, -0.7880859375, -0.66455078125, -0.9052734375, 1.6328125, 1.4912109375, 0.060302734375, 0.11053466796875, 1.9365234375, -0.92724609375, 0.54150390625, -0.36962890625, -0.3544921875, 0.5341796875, 1.478515625, -0.6708984375, -1.5107421875, 1.615234375, 0.278564453125, 0.9189453125, -0.058868408203125, 1.67578125, 0.99267578125, 0.8466796875, -0.56201171875, -0.50146484375, -2.158203125, 0.02008056640625, 0.62255859375, -0.89501953125, -0.5625, -0.259765625, -1.3505859375, 0.5732421875, -0.62890625, 0.67578125, 2.693359375, 0.305419921875, 1.037109375, -1.275390625, 1.0966796875, 1.6591796875, 2.306640625, -0.708984375, -0.8310546875, -0.03216552734375, 1.3623046875, 1.2060546875, -1.4072265625, 0.32373046875, 0.2225341796875, -0.54345703125, -1.3857421875, -0.5986328125, -0.6767578125, 0.77685546875, -0.55517578125, 0.79150390625, -0.297119140625, 0.81591796875, -1.521484375, 0.953125, -0.454833984375, -1.552734375, -0.22705078125, -0.1697998046875, -0.0110626220703125, -1.623046875, -1.2373046875, -0.82568359375, -0.1561279296875, -0.66845703125, 0.49267578125, -0.6904296875, 1.9755859375, -1.001953125, 0.1669921875, 0.404052734375, 1.2568359375, 0.057830810546875, -1.2900390625, 0.60986328125, 1.6181640625, 0.5458984375, -1.7724609375, 0.90869140625, 0.413330078125, 0.277587890625, 0.345947265625, -0.42333984375, -0.37548828125, -0.47998046875, -1.3310546875, -1.3037109375, -0.315185546875, -0.5283203125, 1.029296875, -0.78759765625, 0.283203125, -1.26171875, 0.3671875, 0.040374755859375, 2.302734375, -0.490234375, -0.447998046875, 0.38330078125, 0.315185546875, 2.103515625, -0.401611328125, 1.267578125, -1.0390625, -0.48828125, -2.09765625, 1.228515625, -0.24853515625, -0.2215576171875, -0.0797119140625, -0.35791015625, 0.8623046875, -0.442626953125, -1.181640625, 1.0400390625, -0.08447265625, -1.0146484375, 0.94677734375, 2.033203125, -0.501953125, -1.2822265625, -1.7958984375, 1.3046875, -0.8310546875, -0.1007080078125, 0.53857421875, -0.04071044921875, 0.4921875, -0.1102294921875, 0.0252685546875, -0.72705078125, 3.19921875, -0.91845703125, -0.59326171875, -1.19140625, 0.299560546875, 0.383544921875, 1.6474609375, -0.9287109375, 1.0078125, -0.396728515625, -0.5322265625, 0.408203125, 0.5009765625, -0.8671875, 2.3125, 0.98291015625, 0.493408203125, 0.48779296875, -1.01171875, 0.88916015625, 0.345458984375, 0.11627197265625, 1.6396484375, -0.93310546875, -0.2205810546875, -0.46337890625, -0.87451171875, 0.57080078125, -0.86865234375, -0.0015363693237304688, 0.87646484375, -1.005859375, -0.25439453125, -0.12353515625, -0.68017578125, -0.13623046875, 0.71875, 0.87451171875, -1.5546875, 1.439453125, 1.6875, 0.859375, -1.126953125, -0.7177734375, -0.88037109375, -1.12109375, 0.927734375, -0.34619140625, 0.50634765625, 0.2939453125, -0.1907958984375, 0.7275390625, -0.779296875, -0.77294921875, 0.51220703125, 0.2132568359375, 0.1837158203125, 1.0576171875, -0.73583984375, -1.859375, 0.29833984375, -0.07550048828125, 1.4921875, 0.5244140625, -0.22607421875, -0.366455078125, -0.28125, -0.43798828125, 0.3212890625, 0.3115234375, 0.2369384765625, -1.5234375, 0.1561279296875, -1.1318359375, 0.61279296875, 0.556640625, -0.00033926963806152344, 0.951171875, 0.395263671875, -0.2900390625, -0.97998046875, 0.99853515625, -0.3896484375, 0.174072265625, 0.25927734375, -1.1826171875, 0.89111328125, 0.2958984375, -0.39306640625, -1.876953125, -0.2308349609375, 0.80029296875, -1.521484375, -0.2154541015625, -0.22607421875, 0.52734375, -1.365234375, -0.2626953125, -1.1435546875, -0.146484375, 0.97021484375, 0.017822265625, 0.291015625, 0.025848388671875, -0.77685546875, 1.8994140625, -1.0732421875, -0.34521484375, -0.5771484375, -1.328125, 1.521484375, 0.59326171875, 0.248291015625, 0.9150390625, 0.017578125, -0.49609375, -0.9287109375, -1.4052734375, -0.33251953125, 1.5634765625, 0.2110595703125, 0.6904296875, 0.40869140625, -0.049591064453125, 0.55712890625, -1.1328125, -0.312744140625, -0.505859375, -0.06414794921875, 1.267578125, -1.0166015625, -1.7333984375, 0.76513671875, 0.81884765625, 1.3720703125, -1.2470703125, -0.62109375, 1.7734375, -1.48828125, -0.392822265625, 0.0908203125, 0.1883544921875, 0.76171875, 0.203369140625, 0.13818359375, -0.0810546875, -0.76025390625, 1.765625, -0.7001953125, -1.056640625, 0.06683349609375, 0.24462890625, 0.88623046875, 1.6806640625, 0.69482421875, -0.203125, -0.8271484375, -0.444091796875, 0.6865234375, -1.12890625, 0.166259765625, -0.67919921875, -0.62451171875, -0.63818359375, -0.35302734375, 0.99951171875, 0.255615234375, -0.67626953125, -0.5458984375, -0.806640625, -0.42626953125, -0.311767578125, 0.72119140625, 0.63232421875, -1.2578125, 1.064453125, -0.32666015625, -1.111328125, -0.87353515625, -0.96826171875, -0.471435546875, -1.232421875, 0.64013671875, -0.8583984375, -1.0712890625, 0.2344970703125, -0.57666015625, 0.437744140625, -0.5087890625, 0.1275634765625, -0.90087890625, -1.2587890625, -0.165771484375, -0.55224609375, -0.269287109375, -1.744140625, -0.2474365234375, -0.9482421875, -1.0478515625, -0.7919921875, 1.6298828125, -1.0185546875, 0.395263671875, -0.927734375, 0.14208984375, -0.6845703125, 0.5361328125, 1.3955078125, -0.89111328125, 0.306640625, 0.65234375, 0.408447265625, 0.29443359375, 0.378662109375, -1.5068359375, -0.48046875, -0.453125, -1.6474609375, 1.7197265625, -1.443359375, 0.26123046875, -0.372802734375, -1.01953125, 2.2734375, -0.1981201171875, 0.08978271484375, -0.5986328125, 1.44140625, -0.06573486328125, -0.67138671875, 0.2109375, -0.049957275390625, -0.2529296875, 0.381103515625, 0.28955078125, 0.421142578125, 1.0048828125, -0.52734375, -0.5419921875, 0.5244140625, 0.38037109375, -0.78662109375, 0.57373046875, -0.7353515625, 1.9482421875, 1.7275390625, 0.89794921875, -0.3525390625, 0.8994140625, -0.44970703125, -0.145263671875, -0.479248046875, 0.0030002593994140625, 0.2548828125, -2.740234375, -0.45263671875, 1.15234375, 0.89013671875, -0.39697265625, -1.3876953125, -0.8505859375, 0.150634765625, 0.12451171875, 1.42578125, -0.419677734375, 0.01898193359375, 0.49755859375, -0.235595703125, -0.1259765625, 0.0203857421875, 0.206787109375, 0.1201171875, 1.1484375, -0.5517578125, -0.908203125, 0.81591796875, -0.56884765625, 0.00943756103515625, -0.6962890625, 1.03125, 1.7041015625, -0.435791015625, -0.403564453125, -1.9755859375, 0.464599609375, -0.1258544921875, -1.24609375, -0.54736328125, 0.1507568359375, 0.39697265625, 1.5322265625, -0.0693359375, 0.744140625, 0.71630859375, 0.958984375, 1.2529296875, 1.0517578125, 0.319580078125, 0.98828125, -1.2119140625, -0.458251953125, 1.4501953125, 1.3408203125, 1.6396484375, -0.1036376953125, 0.486572265625, 1.115234375, 0.6416015625, -1.6640625, -0.1719970703125, 1.40625, 0.8095703125, 0.978515625, 1.8896484375, 0.73828125, 2.177734375, -0.6416015625, 0.74365234375, -0.97265625, -1.9521484375, 0.28369140625, 1.8291015625, 0.28466796875, 1.1904296875, -1.4384765625, 1.1044921875, 0.29052734375, 0.3154296875, -0.8486328125, -0.413330078125, -0.27294921875, -0.56005859375, -0.5400390625, -0.51806640625, -0.275146484375, -0.466552734375, 2.109375, -0.7255859375, -1.080078125, -1.4775390625, 0.333251953125, -0.17822265625, -0.5771484375, -0.361572265625, 1.673828125, -0.59228515625, -1.2744140625, 0.82958984375, -1.7529296875, 0.60888671875, -2.513671875, 1.4013671875, -0.19482421875, 0.78662109375, -2.390625, -1.03515625, 0.0001913309097290039, 0.31982421875, 0.951171875, -0.88330078125, -0.37451171875, -0.365478515625, 1.154296875, -0.728515625, 1.2255859375, 0.197021484375, 1.4306640625, -0.1171875, -1.724609375, 0.40380859375, 1.8935546875, -0.87744140625, -1.43359375, 0.6376953125, -0.96435546875, -0.544921875, 0.5986328125, -0.179443359375, 0.3837890625, 0.1651611328125, -0.93359375, -1.62109375, -1.9248046875, -1.2861328125, 0.72119140625, -0.158447265625, 1.7001953125, -0.587890625, -1.0498046875, 0.238037109375, 0.76025390625, -2.009765625, -1.0810546875, 0.133056640625, 0.9052734375, 0.3427734375, 1.4404296875, 2.037109375, -1.2724609375, 0.6416015625, -1.646484375, 0.014984130859375, 0.2568359375, -1.0634765625, -1.11328125, -1.1630859375, 0.26953125, -0.3095703125, -0.1749267578125, 0.5439453125, -0.00902557373046875, -0.5888671875, -0.1015625, 0.5302734375, -0.85107421875, 1.1142578125, -0.357666015625, 2.220703125, 0.07684326171875, -0.7138671875, -1.21875, 0.006587982177734375, 0.254150390625, 1.62109375, -0.286865234375, 1.2412109375, 0.23583984375, -0.0030841827392578125, 1.16796875, -1.541015625, 1.0419921875, 0.271240234375, -1.92578125, -0.2296142578125, -0.7978515625, 0.325927734375, 1.193359375, 0.646484375, 0.1290283203125, -2.390625, -0.38818359375, -1.3251953125, -0.1600341796875, -0.427978515625, -1.0712890625, -0.7412109375, 0.187744140625, -0.5986328125, 0.219482421875, 2.263671875, 0.9658203125, -0.94189453125, -0.87451171875, -0.3056640625, -0.541015625, 1.0751953125, 0.52490234375, -0.83740234375, 0.2037353515625, -0.6181640625, -0.10443115234375, -0.0005497932434082031, 0.47314453125, 0.3251953125, 0.94921875, -0.9677734375, -1.712890625, -0.478271484375, 0.0177154541015625, 0.292724609375, -0.70166015625, -0.71337890625, 0.69921875, 1.421875, -0.03668212890625, -0.52392578125, -1.44140625, 0.58544921875, 0.85107421875, -2.111328125, 0.065673828125, -0.9482421875, -0.335205078125, 1.427734375, 0.27783203125, -1.8408203125, 0.447998046875, 0.5771484375, 0.353515625, 0.4521484375, 0.45703125, -0.61328125, 0.8681640625, 0.54052734375, 0.2056884765625, -0.2205810546875, 1.09765625, -1.826171875, -1.1435546875, 0.0706787109375, 0.4072265625, 0.1610107421875, 1.3359375, -0.3798828125, 2.845703125, -0.2130126953125, 0.388916015625, -0.20849609375, -1.7666015625, 0.52685546875, -0.96875, -0.4931640625, 0.1019287109375, 1.0205078125, -0.77783203125, -1.67578125, 0.5888671875, 0.0203399658203125, -1.1337890625, -1.9619140625, 0.56201171875, -1.24609375, 1.0654296875, 0.058074951171875, -0.222900390625, 0.01557159423828125, -0.90966796875, 1.3837890625, -1.08984375, -0.68994140625, -1.2060546875, 0.07806396484375, 0.1407470703125, -1.0556640625, 0.56689453125, 1.951171875, -0.68701171875, 0.10125732421875, -1.4296875, 0.13134765625, 1.794921875, -0.46826171875, -2.087890625, -0.55712890625, -1.8671875, -1.427734375, 0.6103515625, 0.880859375, -0.1429443359375, 0.873046875, 0.732421875, 0.482177734375, -0.68115234375, -1.1923828125, -1.7392578125, -1.4033203125, 0.354736328125, -0.340576171875, -0.051483154296875, -0.06109619140625, -2.0390625, 2.87109375, -0.277099609375, -0.07037353515625, -1.6552734375, -1.4228515625, -0.73193359375, -0.09417724609375, 0.0260162353515625, 0.0927734375, 0.6455078125, 0.9140625, 0.98388671875, 0.2401123046875, 0.4501953125, 1.3505859375, 1.2119140625, 0.4658203125, 0.473876953125, -0.7265625, 1.11328125, -0.336181640625, 1.986328125, 0.3037109375, -0.44775390625, 1.998046875, 0.6533203125, -0.6591796875, -1.189453125, 0.62451171875, -0.7978515625, 0.452392578125, -0.353515625, 0.845703125, -0.255859375, -0.56396484375, 0.5478515625, -0.360107421875, -0.3037109375, -0.8466796875, -0.2239990234375, -0.580078125, 1.33984375, -0.1416015625, 0.67626953125, 1.0166015625, -0.31103515625, 0.8828125, 1.6513671875, -0.10333251953125, 1.1708984375, -0.01473236083984375, 0.276123046875, 0.84619140625, -1.33984375, -0.466064453125, 2.1171875, 0.300048828125, -0.379150390625, -1.0205078125, -0.64599609375, 0.09722900390625, -0.489501953125, 0.248779296875, 1.1474609375, 0.1236572265625, 0.57861328125, 1.7138671875, 0.51318359375, 0.3251953125, 0.865234375, 0.771484375, 0.76513671875, 0.325439453125, 0.64892578125, -1.080078125, -0.03643798828125, -0.927734375, 0.9287109375, -0.351806640625, 1.3564453125, -0.08306884765625, -0.61474609375, 0.2354736328125, 0.935546875, 1.203125, -0.1634521484375, 0.0614013671875, 0.4560546875, 0.20556640625, -2.369140625, 0.89404296875, -1.1123046875, 1.7109375, -0.52734375, -0.0374755859375, -0.1280517578125, 0.174072265625, 0.88916015625, 1.0166015625, 0.89892578125, -1.244140625, 0.1297607421875, 0.325927734375, 2.3125, -0.22802734375, -0.490966796875, 1.6708984375, -0.517578125, 0.312255859375, -2.10546875, -1.8681640625, 0.3115234375, -0.7431640625, 0.38818359375, -0.8818359375, -1.8681640625, 0.318359375, 0.50390625, 0.369140625, -0.8740234375, -0.814453125, -0.479248046875, -0.95556640625, -0.81640625, -1.64453125, -1.09765625, -0.00960540771484375, 0.53076171875, 0.1253662109375, -0.875, 1.0400390625, -0.21923828125, -0.837890625, 0.8291015625, 0.57421875, -0.6298828125, 0.8193359375, -1.630859375, 0.3759765625, 2.25, -0.9794921875, -0.63232421875, -1.2060546875, -1.828125, -0.69580078125, -0.2325439453125, 0.939453125, -0.4375, -1.5859375, -1.25390625, -0.198974609375, 0.19873046875, 0.64404296875, -0.76416015625, -0.10723876953125, -0.195068359375, 0.2066650390625, -0.693359375, 0.51416015625, -0.0137786865234375, -0.6337890625, -0.316650390625, 0.05364990234375, 0.343017578125, 1.0927734375, 1.146484375, -2.26171875, 0.4443359375, -0.81787109375, 1.9970703125, 0.669921875, -0.646484375, 0.22900390625, 0.39453125, -1.048828125, -0.2705078125, -0.951171875, 0.49609375, -0.92724609375, 0.74853515625, -1.9765625, -2.630859375, 0.2469482421875, -0.154052734375, -0.6005859375, 2.072265625, -1.0244140625, 1.0009765625, -0.64892578125, 0.286865234375, -0.55029296875, 1.392578125, 0.7158203125, 1.1865234375, -1.8408203125, 0.908203125, 0.09149169921875, -0.646484375, -0.93701171875, 0.7802734375, 1.470703125, -0.673828125, 0.83984375, 2.072265625, 0.025604248046875, 1.1142578125, 0.11663818359375, 1.1943359375, 1.4140625, 0.367919921875, -2.189453125, -2.220703125, 0.833984375, -0.1910400390625, 0.417236328125, -1.4892578125, -2.353515625, 0.82080078125, -0.305419921875, 0.1639404296875, 0.130859375, -0.97216796875, 1.453125, -0.221923828125, 0.439453125, -1.64453125, -0.09210205078125, 0.80908203125, 0.3515625, -0.5283203125, 1.0849609375, -0.274169921875, -0.73828125, 1.056640625, 0.52685546875, -0.87060546875, -0.904296875, 0.984375, 0.4130859375, -1.181640625, -1.361328125, -0.1407470703125, 0.482421875, -0.38623046875, 0.10723876953125, 1.841796875, -1.0615234375, 1.142578125, -0.56787109375, -1.83984375, -0.66943359375, 0.1748046875, 0.298583984375, -0.163330078125, 1.4873046875, 0.354248046875, -0.32421875, -0.78955078125, 0.12103271484375, 0.404541015625, 0.447998046875, -1.5341796875, 0.98193359375, 0.6962890625, -1.712890625, -0.70458984375, -0.72607421875, 0.150146484375, 0.5341796875, -2.50390625, 0.2459716796875, -0.83203125, -0.2352294921875, 0.6943359375, -1.12109375, 0.83984375, 1.2373046875, 1.353515625, 0.6572265625, -0.048553466796875, -0.283203125, -0.156982421875, -0.0943603515625, 1.3310546875, -0.431640625, 1.71484375, 0.410888671875, 1.73046875, -1.0927734375, 1.2646484375, 1.9736328125, -0.1796875, -1.07421875, -0.2080078125, 0.5068359375, 0.199951171875, 1.080078125, 0.3408203125, -1.26171875, 0.72021484375, 0.05364990234375, 0.38720703125, -0.43701171875, 1.3935546875, -1.1611328125, 0.578125, 1.029296875, 0.71875, 0.84765625, -0.87841796875, -0.5791015625, 0.210693359375, -0.56982421875, 1.271484375, -1.3349609375, -0.455810546875, -1.9794921875, -0.085693359375, -0.67529296875, 0.5888671875, 1.25, 0.90283203125, -0.6953125, 2.330078125, 0.308837890625, -1.916015625, -0.66796875, -1.125, 1.8466796875, -0.322509765625, 0.55810546875, -0.7421875, -1.501953125, 1.46484375, -0.095703125, 0.234130859375, 1.0146484375, 0.14306640625, 0.272705078125, -0.615234375, 0.50390625, 1.8291015625, 0.259521484375, -0.367431640625, 0.82080078125, -2.197265625, -0.375732421875, -0.277587890625, 0.736328125, 1.1015625, 0.6611328125, -0.61865234375, -0.467529296875, -0.95654296875, 0.73046875, 0.8642578125, -0.99853515625, -1.970703125, -1.11328125, 0.87353515625, -1.11328125, -0.8115234375, -1.5244140625, -1.0166015625, 1.4501953125, 0.367431640625, -0.69287109375, -1.3701171875, -0.1788330078125, -1.765625, 1.763671875, -1.185546875, -0.69580078125, 1.326171875, 0.638671875, -0.857421875, 1.267578125, -1.666015625, -0.1402587890625, 0.0731201171875, -2.01953125, 0.75830078125, -1.5361328125, -0.8251953125, -0.08038330078125, 0.17626953125, -0.7841796875, 0.859375, 0.603515625, -0.375, 1.0458984375, -0.568359375, 0.1517333984375, 0.83740234375, 1.04296875, -0.1885986328125, -0.11474609375, 0.2178955078125, -1.18359375, 1.0361328125, 0.1224365234375, 1.2744140625, -0.90869140625, 1.3388671875, 0.2208251953125, 0.25341796875, -0.0255279541015625, -0.0218658447265625, 0.2381591796875, 0.25634765625, -1.0322265625, 0.86474609375, 1.455078125, 0.06219482421875, -0.0545654296875, 0.9453125, 1.08203125, 0.03741455078125, -1.484375, -0.77099609375, 1.3759765625, 1.732421875, 1.6806640625, 1.6201171875, -1.0458984375, -1.060546875, 0.2498779296875, -1.1318359375, 0.7294921875, 0.09478759765625, 1.2138671875, 0.96142578125, -1.1689453125, -0.9453125, -1.044921875, 0.26611328125, -0.63525390625, -1.3935546875, 0.20947265625, -0.58349609375, -0.22314453125, -1.998046875, -0.04412841796875, -0.2264404296875, 0.296630859375, -0.705078125, -1.1630859375, 1.9873046875, 1.326171875, -0.82666015625, -1.6318359375, -0.8515625, 0.00746917724609375, -0.492919921875, 0.62255859375, 0.2447509765625, 0.497314453125, -1.3779296875, -2.669921875, 0.309814453125, -1.1943359375, 1.6826171875, -0.4423828125, -0.8505859375, -1.8212890625, -0.63134765625, 0.75390625, 0.2283935546875, -0.787109375, 0.339599609375, 0.48291015625, -0.7890625, 0.828125, 0.7587890625, -0.82275390625, 1.322265625, -0.96875, 0.08843994140625, 0.34375, 0.19287109375, 0.646484375, -0.880859375, -0.07952880859375, 0.07025146484375, -0.25048828125, 0.141357421875, 0.66015625, -2.072265625, 1.62890625, 0.12225341796875, -1.6083984375, 1.40625, -0.9052734375, 0.178466796875, 0.96923828125, 0.4296875, -0.2152099609375, -0.6767578125, -0.054473876953125, -1.7548828125, -0.6630859375, -1.94140625, 0.80419921875, 0.1502685546875, 0.032989501953125, -1.310546875, 0.2039794921875, -0.47509765625, -1.279296875, -1.4501953125, 0.148681640625, -0.396484375, -1.0107421875, 1.724609375, 1.06640625, 0.5146484375, 0.615234375, 1.453125, -1.3212890625, -0.27001953125, -0.005138397216796875, -0.84033203125, 0.111328125, 0.2130126953125, 2.19140625, 0.55859375, -0.5263671875, 0.38330078125, -1.0693359375, 0.50927734375, -0.23486328125, 1.576171875, 0.2396240234375, -1.365234375, -1.5693359375, 1.978515625, -0.1795654296875, 0.7314453125, -0.2283935546875, -0.135498046875, -1.36328125, -0.08453369140625, -0.76513671875, -1.0712890625, 1.83984375, -0.048919677734375, 0.21533203125, 0.32470703125, -0.0677490234375, -0.62255859375, 0.1348876953125, 0.255859375, -0.9912109375, -0.11920166015625, 0.284423828125, -0.7451171875, -2.5625, 2.73828125, 1.4677734375, -0.307861328125, 0.2060546875, -0.60400390625, -0.0011539459228515625, 1.3134765625, -0.1383056640625, 1.099609375, -0.035736083984375, 1.42578125, 0.416259765625, -1.6533203125, 0.92578125, 1.1220703125, -0.106689453125, -0.47119140625, 1.421875, -1.87109375, -0.3623046875, -1.5419921875, 1.85546875, -1.0048828125, -0.94482421875, -0.80859375, 0.326171875, 0.2398681640625, 0.03851318359375, 0.51416015625, 1.052734375, 0.275146484375, 0.40087890625, -0.0266876220703125, 0.1627197265625, -1.49609375, -1.7685546875, 1.5087890625, -1.685546875, -0.5869140625, -1.2744140625, -0.91943359375, -1.6328125, -0.1798095703125, -1.0498046875, -0.425048828125, -0.70654296875, -0.61181640625, -2.34375, 0.0889892578125, -0.10345458984375, -1.275390625, -1.2744140625, 1.537109375, 0.84228515625, -1.126953125, -1.7373046875, -1.4111328125, 1.6083984375, -0.6708984375, -0.52734375, 0.90478515625, -1.1767578125, -1.2802734375, -0.085693359375, 0.0943603515625, 0.55029296875, 0.4150390625, 0.1507568359375, 1.5458984375, -0.73486328125, 1.4609375, 1.013671875, 1.59375, 1.193359375, -1.0703125, -1.6533203125, -0.461181640625, -0.08074951171875, 1.4736328125, 1.548828125, 1.0439453125, 0.411865234375, 2.017578125, -1.3076171875, -1.84375, 0.08154296875, -1.33984375, -0.171630859375, -0.80859375, -0.546875, -1.162109375, -1.5419921875, 1.072265625, 0.5, -1.2412109375, 1.5712890625, -0.129150390625, 0.09381103515625, 0.63232421875, 0.422607421875, -1.1162109375, -1.7080078125, 0.389892578125, -1.2802734375, 0.9541015625, -0.228759765625, 2.123046875, 0.0567626953125, -1.056640625, 1.7177734375, -1.1171875, 0.48046875, 0.004245758056640625, -0.564453125, 0.75927734375, -0.51806640625, 1.908203125, -0.22802734375, 0.2344970703125, 0.54736328125, -0.05487060546875, 0.30517578125, 0.393798828125, 1.4443359375, 0.85791015625, 1.501953125, 0.15087890625, -0.433837890625, -0.96923828125, 0.07342529296875, 0.21484375, 0.59716796875, -0.232421875, 1.517578125, -0.6767578125, -0.63623046875, -1.7587890625, -0.6142578125, 0.029541015625, -0.59912109375, 0.24609375, 0.1458740234375, -1.748046875, -1.48828125, -0.07257080078125, 0.783203125, -0.219970703125, 0.267333984375, -2.162109375, -1.16015625, -0.498291015625, 0.15234375, -0.82080078125, -0.3779296875, 1.1025390625, -0.037017822265625, -1.0869140625, 1.884765625, -0.5693359375, 0.583984375, 1.6484375, -0.51416015625, 0.525390625, -1.3310546875, -0.8291015625, 0.2374267578125, 1.1884765625, -2.15625, 2.771484375, -0.396240234375, 0.54150390625, -1.3857421875, 0.71435546875, -0.2607421875, 0.1295166015625, 1.732421875, 1.33203125, 0.429443359375, -0.7685546875, 0.07177734375, -0.54638671875, -0.5537109375, -0.73193359375, -1.09765625, -0.6689453125, 0.6572265625, -0.96923828125, -1.4169921875, -1.9365234375, -0.415771484375, 0.3779296875, -0.55517578125, -0.348876953125, 1.16796875, -0.7373046875, -0.802734375, -0.28271484375, -0.73583984375, -0.01172637939453125, 0.62890625, 1.34375, 1.046875, 0.2418212890625, 0.006313323974609375, 0.0184173583984375, -0.1834716796875, -0.1187744140625, 1.0654296875, -0.322021484375, -0.39697265625, -0.5048828125, 2.6796875, -0.8359375, -0.73046875, -2.001953125, 0.263671875, 0.07830810546875, 1.0146484375, 0.87939453125, 0.1650390625, 1.8818359375, -1.7177734375, -0.603515625, -0.2022705078125, 1.2314453125, -0.76806640625, -1.2734375, 0.83154296875, -0.39404296875, 0.059478759765625, -2.376953125, -0.8916015625, -0.07122802734375, -1.056640625, 1.9404296875, -1.2529296875, 0.5283203125, 0.76318359375, -0.279296875, -2.10546875, 1.939453125, 0.019775390625, -0.6767578125, -0.822265625, -0.00534820556640625, -0.2303466796875, 0.951171875, -0.1302490234375, -0.11248779296875, -0.7314453125, -0.61865234375, -1.6123046875, -0.452880859375, -0.72705078125, 1.2802734375, 0.64501953125, -0.53955078125, 1.0625, -0.857421875, -0.453125, -2.162109375, -1.392578125, -0.8173828125, -0.3876953125, -0.689453125, 0.344482421875, 0.60009765625, 0.1663818359375, 0.062469482421875, 0.1728515625, 1.291015625, 0.09613037109375, -1.11328125, -0.45263671875, -0.3955078125, -0.69580078125, 0.50634765625, -1.06640625, 0.32373046875, 0.1650390625, 0.1483154296875, -0.947265625, 0.4501953125, 0.32763671875, 0.60302734375, 1.6083984375, 0.55908203125, 0.033294677734375, -1.1142578125, 0.97998046875, 1.5234375, -1.6572265625, -0.030029296875, 0.73779296875, 0.25341796875, 0.35205078125, -1.560546875, -0.51220703125, -1.8642578125, 0.869140625, 1.978515625, 1.0087890625, -1.1337890625, 0.78857421875, 0.8466796875, -1.1005859375, -0.1890869140625, -1.06640625, -0.7119140625, 0.310791015625, -0.07122802734375, -0.198974609375, 0.42041015625, 0.47607421875, -0.50244140625, -1.763671875, -0.34912109375, 1.818359375, 1.283203125, -1.427734375, 0.9765625, 1.4443359375, -0.275390625, 0.298095703125, 0.25634765625, 0.0017309188842773438, 1.3984375, 0.440673828125, 0.11248779296875, 0.33203125, 0.157470703125, -0.89306640625, -1.0078125, 0.833984375, 2.306640625, -0.52099609375, 1.29296875, -0.2315673828125, -0.64306640625, 2.400390625, -0.77001953125, -1.1171875, -1.4892578125, 0.33984375, -0.43798828125, -1.037109375, -0.3896484375, -1.125, -0.607421875, 0.47216796875, -0.284912109375, 1.0927734375, 0.552734375, 1.1005859375, -1.501953125, -0.0340576171875, 0.78662109375, -0.53466796875, 0.0303192138671875, 2.14453125, -0.6806640625, -0.0711669921875, 0.287109375, -1.5537109375, -0.042877197265625, -1.328125, 0.28564453125, 0.77197265625, 0.06732177734375, 0.377197265625, 1.27734375, -2.248046875, 0.03753662109375, -0.1849365234375, -1.21484375, 0.369140625, -0.2349853515625, 0.357421875, -0.1700439453125, 0.75634765625, 1.9052734375, 0.86865234375, 0.01056671142578125, 0.3662109375, -1.65625, -0.9296875, 2.041015625, -0.0927734375, -1.0966796875, -1.3876953125, -0.48974609375, 0.450439453125, 0.10968017578125, -1.22265625, 2.880859375, -0.240478515625, 2.048828125, -0.5126953125, -1.1923828125, -0.3115234375, 0.59765625, 0.326904296875, -0.07904052734375, 0.1015625, -0.349853515625, 0.31005859375, -0.71875, -1.76171875, 0.9306640625, -1.4560546875, -0.404296875, -1.14453125, -0.273681640625, 0.3974609375, 0.47216796875, 0.4521484375, -1.7021484375, -0.9794921875, -1.4384765625, -1.23046875, 1.16796875, 0.57861328125, -1.15625, 0.487060546875, -0.1312255859375, -0.6806640625, 0.7001953125, -0.73388671875, 0.085693359375, -0.72802734375, -0.154052734375, 0.0848388671875, -0.2294921875, -0.28271484375, 0.053863525390625, 1.1337890625, 1.81640625, 0.33447265625, 0.50830078125, 2.810546875, -0.290771484375, -0.2169189453125, 1.232421875, 1.654296875, -0.260498046875, 0.06182861328125, 0.41259765625, -0.445068359375, 0.373291015625, -0.73388671875, 0.82373046875, -0.77880859375, 0.5712890625, 1.5078125, -0.287841796875, -2.5703125, 2.798828125, -0.74267578125, 1.6162109375, 0.381591796875, -0.107421875, -0.162109375, 1.5654296875, 0.67919921875, -0.33740234375, -1.150390625, -0.5625, -0.168212890625, -1.7529296875, -0.002849578857421875, -0.013336181640625, 1.552734375, 0.479248046875, -0.272216796875, 0.091796875, 0.5625, -1.7939453125, -0.98193359375, -1.1171875, 0.2147216796875, -2.060546875, -0.25146484375, -0.2293701171875, 0.20849609375, -1.0478515625, 1.1875, -0.134765625, -0.92431640625, 0.12646484375, 1.1416015625, -0.073974609375, -0.7607421875, -0.986328125, 2.029296875, 0.1162109375, -0.093994140625, -0.2142333984375, 1.1650390625, -0.49853515625, -1.0341796875, 0.26318359375, -0.9326171875, -1.9189453125, -0.63037109375, -2.1015625, 0.70166015625, -1.21484375, 0.56298828125, -0.396484375, -1.138671875, 1.001953125, -0.7021484375, 0.1226806640625, -0.56787109375, 0.1844482421875, -1.3310546875, 0.5771484375, 1.1162109375, 0.208984375, -0.011505126953125, -1.8486328125, -2.58984375, -1.06640625, 0.029937744140625, -0.5283203125, -0.58935546875, -0.481689453125, -0.623046875, -0.0294189453125, -1.517578125, 1.51171875, -0.37451171875, 2.291015625, 1.12890625, -1.6083984375, -0.020751953125, -0.7373046875, 0.65673828125, -0.07647705078125, -0.5439453125, -1.505859375, -0.06549072265625, 0.355224609375, 0.91748046875, 1.37109375, -0.5888671875, 1.15234375, 0.8291015625, -0.0936279296875, 0.2366943359375, 0.445068359375, 0.018341064453125, -0.36962890625, 2.3359375, -0.486083984375, 0.90576171875, -0.410400390625, 0.22265625, 0.51904296875, 0.421875, 1.3232421875, -1.6337890625, 1.2763671875, -0.56298828125, -0.71923828125, -0.72021484375, -2.158203125, -0.73486328125, -1.4462890625, 0.0689697265625, 0.923828125, -0.4296875, -1.5048828125, -1.9560546875, -1.9521484375, 0.491943359375, 1.271484375, -0.52490234375, 2.33203125, -1.9326171875, 1.1787109375, -0.26953125, -0.2420654296875, -1.6279296875, 1.966796875, 0.67626953125, -1.3828125, 0.86328125, 1.740234375, 0.76904296875, -0.205078125, 0.089111328125, 0.08221435546875, -1.2705078125, 0.76904296875, 0.07708740234375, 1.85546875, -0.45166015625, 0.82470703125, 0.50927734375, -0.65625, -0.477294921875, -0.6328125, 0.4345703125, -1.955078125, 1.0771484375, -0.282958984375, 0.61962890625, 0.65869140625, 0.231689453125, 0.92919921875, 0.75537109375, 0.94873046875, -1.0, 0.342041015625, -0.70849609375, -0.63916015625, -1.0078125, -1.5205078125, 0.365478515625, -1.1513671875, 0.8203125, -1.4111328125, -0.7109375, -2.810546875, -1.4423828125, -0.857421875, -0.394287109375, 1.1416015625, 0.953125, -0.308837890625, 0.2388916015625, -0.1666259765625, -1.1708984375, -0.60791015625, 0.0098114013671875, 0.6943359375, 0.18505859375, 1.048828125, 1.1572265625, 1.06640625, 0.7939453125, -1.921875, 1.5517578125, 0.438720703125, -0.63623046875, -0.63134765625, -0.92626953125, 1.1328125, -1.30078125, 1.3828125, -0.8759765625, -0.169677734375, 0.7900390625, 0.79931640625, -0.429931640625, 0.09149169921875, -0.452392578125, 0.767578125, 1.1337890625, -0.91015625, 0.0156402587890625, 0.568359375, -1.634765625, 0.529296875, -1.1416015625, -1.3154296875, 0.205078125, 0.7001953125, 0.278076171875, -0.86962890625, -0.4970703125, -0.2373046875, 0.34326171875, -0.49609375, -0.787109375, 1.765625, 1.15625, 1.1455078125, 0.833984375, -0.9228515625, -0.063232421875, -0.357421875, 0.953125, -1.642578125, 0.3662109375, 0.9375, 0.1241455078125, 0.2005615234375, 0.6435546875, -0.426513671875, 1.14453125, -0.11083984375, 0.1573486328125, -0.0005292892456054688, 0.127197265625, 1.10546875, -0.271728515625, -1.48828125, -0.287109375, -0.40234375, 1.0458984375, -1.2841796875, -0.77880859375, 1.103515625, 0.5849609375, -0.424072265625, -0.306640625, 1.40625, -0.061187744140625, 0.4189453125, -0.08966064453125, -1.0107421875, -0.401611328125, -0.483642578125, 0.256103515625, 3.41015625, 0.439208984375, -0.5634765625, 0.12158203125, 0.640625, -0.904296875, 0.89501953125, 0.94384765625, 1.001953125, 1.7197265625, -0.52392578125, 0.6640625, 0.47314453125, 1.43359375, -0.38134765625, 0.4912109375, 1.19921875, -0.13427734375, -0.5517578125, 0.3974609375, 0.0533447265625, -0.8671875, -0.03582763671875, -0.7041015625, -0.233642578125, -1.2236328125, -1.279296875, -0.576171875, 1.9140625, -1.1357421875, 1.72265625, 1.04296875, -0.6171875, 0.1473388671875, -1.658203125, -0.61181640625, 0.591796875, -0.67529296875, 0.72607421875, -0.57861328125, -0.379150390625, -0.8125, 1.787109375, -0.8896484375, -0.5556640625, 0.78564453125, -1.0869140625, -0.80419921875, -0.759765625, -0.60888671875, 0.7744140625, -0.058013916015625, -0.074951171875, 0.74169921875, -0.6396484375, 0.25634765625, 0.13525390625, -0.79736328125, -1.68359375, 0.654296875, -1.3251953125, -0.2027587890625, 0.45458984375, 1.322265625, -1.5234375, -1.34375, 1.2021484375, 0.97705078125, 1.3330078125, -0.62939453125, 0.39697265625, -0.255615234375, 0.413818359375, 0.403564453125, 1.248046875, -0.54833984375, 1.9296875, 1.865234375, 0.384521484375, 0.85791015625, -0.65087890625, -0.42041015625, -0.56787109375, 1.31640625, 0.389404296875, -0.5732421875, -1.4765625, -0.360107421875, -0.63037109375, 0.379638671875, 0.76171875, -0.60205078125, -1.5361328125, 0.6240234375, 0.7265625, -0.55078125, 0.051544189453125, 0.90283203125, 0.1595458984375, 0.72314453125, -0.84375, 1.0380859375, -0.10858154296875, -1.1494140625, -0.400634765625, 0.0799560546875, -1.236328125, -0.302734375, -2.078125, -1.40234375, -0.609375, 0.6533203125, -0.05712890625, 0.95166015625, 0.2298583984375, -1.765625, -0.302978515625, 0.7939453125, 0.2685546875, -1.15234375, 0.11578369140625, -0.8251953125, 0.32568359375, -0.79638671875, -0.029571533203125, -0.33251953125, 0.9482421875, -0.6298828125, 0.5185546875, 0.028472900390625, -1.591796875, 0.93701171875, -1.4306640625, 1.6376953125, 1.267578125, 1.228515625, 0.462646484375, 1.212890625, 0.46435546875, -0.857421875, 0.14794921875, -0.69970703125, -1.4013671875, -1.1015625, 0.05810546875, -0.64990234375, 1.0009765625, 0.7724609375, 0.06768798828125, 1.125, -1.396484375, 1.1064453125, 0.10302734375, -0.239501953125, -0.37060546875, 0.79541015625, -0.2073974609375, 1.1416015625, -0.6376953125, -0.66650390625, -1.7646484375, -0.51416015625, -0.85888671875, -0.5126953125, 0.68408203125, 2.470703125, -0.7900390625, -0.513671875, 0.970703125, -0.419921875, -1.7939453125, 0.162109375, -1.150390625, 0.99365234375, -0.281005859375, 0.53076171875, -0.71630859375, 0.223876953125, 0.64208984375, 0.46044921875, 0.33447265625, 0.94287109375, -0.1639404296875, -0.09869384765625, -0.65673828125, -0.3564453125, -0.38671875, 0.51513671875, 0.2281494140625, 0.0955810546875, 1.65234375, -0.82373046875, 0.017852783203125, -0.2296142578125, 0.9853515625, -0.5859375, -1.0703125, -1.6220703125, -0.57568359375, 0.10601806640625, -0.93310546875, 0.143798828125, 0.93896484375, -1.1455078125, -1.3623046875, -0.83642578125, -1.2119140625, -0.0180816650390625, 0.75439453125, 0.53173828125, -0.180419921875, 0.68798828125, 0.037811279296875, 0.0927734375, 0.705078125, -0.43994140625, -0.051971435546875, 1.0859375, 0.045623779296875, -0.642578125, 0.0140838623046875, -0.095703125, -0.66748046875, -0.22119140625, 2.087890625, -1.2841796875, 0.568359375, 1.1845703125, 1.931640625, 1.0673828125, 0.84423828125, 1.8486328125, 0.171142578125, -0.91015625, 0.7236328125, -1.6494140625, 0.470947265625, 0.521484375, -0.7099609375, -1.4755859375, -0.0146942138671875, -1.3408203125, -1.3212890625, -1.869140625, -0.328857421875, 1.2431640625, -0.7236328125, -1.1064453125, -0.104248046875, 1.9072265625, 1.2041015625, -1.4892578125, 0.439208984375, 1.3291015625, -0.0731201171875, 1.359375, 0.10321044921875, 0.9755859375, -0.2587890625, -0.892578125, -3.271484375, 0.36083984375, -0.378173828125, 0.1082763671875, 0.277099609375, -1.072265625, -0.73876953125, -0.05242919921875, -0.183349609375, 0.7421875, -0.402099609375, -1.7685546875, -0.69970703125, 1.05859375, 0.6640625, 0.990234375, 0.91650390625, 0.0147705078125, -1.0966796875, 1.580078125, -1.1376953125, -0.83349609375, -0.67431640625, -0.30419921875, -1.41015625, 0.09112548828125, -1.1044921875, -1.2265625, 0.916015625, -0.921875, -0.1705322265625, -0.1483154296875, -2.076171875, 0.1986083984375, 0.2493896484375, 1.2138671875, -0.5341796875, 0.12109375, 0.55517578125, -0.2459716796875, -0.28076171875, 1.0830078125, -2.533203125, 0.7392578125, 1.9248046875, 0.2266845703125, 1.751953125, 1.5185546875, 0.72265625, 1.8544921875, 0.595703125, -1.6748046875, 0.6298828125, -0.76220703125, 0.62109375, 1.904296875, 1.5107421875, 0.0364990234375, 1.400390625, 0.0401611328125, -0.195068359375, 1.1767578125, 0.14306640625, -0.64208984375, -1.1376953125, -1.2744140625, -0.70751953125, 0.9892578125, 0.96044921875, 0.00939178466796875, -0.9072265625, -0.09051513671875, 0.4599609375, 0.94775390625, 2.09375, -0.1884765625, 1.2666015625, 0.91162109375, -2.736328125, 0.4111328125, 0.346435546875, 0.09844970703125, -0.701171875, 1.64453125, -0.533203125, 0.32080078125, -0.69970703125, 0.97021484375, 1.080078125, 1.072265625, -0.779296875, -0.1497802734375, -0.833984375, 0.310546875, 0.97998046875, 1.703125, 2.00390625, 0.7236328125, -0.1629638671875, -1.677734375, -1.1171875, -1.77734375, -1.2900390625, -0.671875, -0.10406494140625, 2.330078125, 0.17578125, 0.333251953125, 0.1622314453125, 0.08941650390625, -0.406005859375, -0.12237548828125, 0.207763671875, -1.572265625, 1.85546875, -0.271240234375, -0.39501953125, 0.77587890625, 1.6357421875, -0.58935546875, -1.3173828125, 2.232421875, -0.2008056640625, -1.5986328125, -2.083984375, -2.716796875, -1.259765625, -2.1484375, 1.5107421875, 0.11944580078125, 0.34326171875, 0.288330078125, 0.7265625, -0.494873046875, 0.87060546875, 0.5810546875, 2.416015625, 1.4755859375, 0.46435546875, -0.08929443359375, -1.66015625, 0.5634765625, 0.69384765625, 0.6669921875, -0.60595703125, 0.01508331298828125, 0.480712890625, 2.16015625, 0.354736328125, -1.458984375, 1.265625, -0.2301025390625, 1.7333984375, 0.38232421875, -0.55126953125, 0.21435546875, 2.0, 0.07110595703125, -2.10546875, 0.64306640625, 0.837890625, 0.1539306640625, 0.71240234375, -0.4482421875, -0.1251220703125, -2.109375, -0.91748046875, 0.583984375, 1.3056640625, -0.10223388671875, -0.513671875, -0.9423828125, -0.21923828125, -0.27880859375, 0.89453125, 0.98388671875, -2.279296875, 2.18359375, -0.274658203125, 0.378173828125, 0.2423095703125, -0.6044921875, 1.083984375, -1.23046875, 0.5859375, -0.5927734375, -0.276611328125, -1.0009765625, -0.225341796875, 0.6083984375, -0.2225341796875, 0.58740234375, 1.5810546875, -1.4990234375, 0.38134765625, -0.44677734375, -0.38427734375, 0.634765625, 1.236328125, -1.171875, -0.408447265625, 1.4345703125, 0.01418304443359375, -0.198486328125, 0.08392333984375, -2.34375, 0.4951171875, -0.69091796875, -0.370849609375, -1.1962890625, 0.78076171875, -0.5048828125, 1.1611328125, -1.59765625, -1.80078125, -0.4599609375, -0.4189453125, 0.74951171875, -0.712890625, -0.26123046875, 0.253173828125, -0.609375, 1.2958984375, 0.62109375, 0.382568359375, -0.8203125, -0.5390625, -0.0810546875, -0.375732421875, 0.26806640625, 0.5830078125, 1.51953125, -1.470703125, -0.8212890625, 1.0556640625, 0.473388671875, 1.828125, 0.314453125, 1.421875, 0.46240234375, -0.416015625, -0.84326171875, -1.353515625, -0.59912109375, -1.591796875, -0.022552490234375, 0.548828125, -0.47119140625, -1.15625, 0.8046875, 2.041015625, 0.0041351318359375, 1.53125, 0.2135009765625, -0.92919921875, -0.1605224609375, -0.7822265625, -1.021484375, -0.1290283203125, 0.4423828125, -1.0771484375, -0.445068359375, 0.11279296875, 1.052734375, 0.15625, 0.71923828125, -0.57177734375, -0.6806640625, 0.2406005859375, 0.14208984375, -2.08984375, -0.12274169921875, -0.83349609375, 1.1689453125, -0.9873046875, -1.2138671875, -1.1494140625, -0.791015625, -0.318115234375, 0.92138671875, -0.387939453125, -0.450439453125, -0.865234375, 1.3505859375, 1.232421875, 0.79248046875, 0.5751953125, 0.306884765625, -1.4580078125, -0.380126953125, 0.23583984375, -0.69140625, -0.40283203125, 0.1795654296875, 0.34130859375, 0.177734375, 1.158203125, -1.37890625, -0.69091796875, -0.26513671875, 0.48681640625, 1.6328125, -0.360107421875, -0.09967041015625, -0.302001953125, -0.161865234375, -1.1220703125, 1.21875, 0.78271484375, -0.607421875, 1.482421875, 0.44384765625, 0.007541656494140625, 0.6435546875, 1.21484375, 0.444580078125, 1.091796875, -0.70849609375, 1.044921875, -0.5947265625, 0.56787109375, 1.37890625, 0.88330078125, -0.35205078125, -0.38134765625, 2.0625, -0.63818359375, -0.68017578125, -0.304931640625, -1.6259765625, -0.83740234375, -1.51171875, -0.51611328125, 0.274169921875, -0.2447509765625, 1.2392578125, -1.017578125, 0.16943359375, 1.283203125, 0.9814453125, -0.69091796875, -2.271484375, -0.3837890625, 0.15380859375, 0.464599609375, 1.2548828125, 0.03228759765625, 0.05352783203125, -0.736328125, 0.026641845703125, -0.89013671875, -0.115234375, -0.806640625, 0.86083984375, 1.234375, 0.94287109375, -2.26171875, 0.82763671875, 0.172119140625, -0.36767578125, -0.12017822265625, -0.90380859375, 1.26171875, 1.49609375, -1.005859375, 1.3759765625, 1.5966796875, -0.51220703125, -1.0693359375, -0.8642578125, -1.1103515625, -1.662109375, -0.35546875, 1.2197265625, 0.040191650390625, 0.06494140625, -1.6982421875, -1.306640625, 0.161376953125, 0.485595703125, -0.4921875, -1.1689453125, 0.77099609375, 0.98828125, 1.2880859375, -0.6162109375, 0.00510406494140625, -2.931640625, 0.79736328125, -0.4736328125, 0.51513671875, -0.34912109375, -1.8857421875, 1.3525390625, 0.662109375, -0.8291015625, 1.1474609375, 0.84716796875, 0.1103515625, -1.5556640625, 1.3447265625, -1.2421875, -1.337890625, 0.58544921875, 0.9521484375, 0.8359375, -0.96728515625, 0.50830078125, -0.2117919921875, -0.6240234375, 1.005859375, 0.498046875, -1.736328125, -1.8994140625, 0.865234375, 0.049774169921875, 0.5302734375, -0.1009521484375, 0.455078125, 0.33056640625, 0.361083984375, 0.45703125, 0.78759765625, -1.4375, -1.8544921875, 0.342041015625, 0.873046875, 1.181640625, -0.71484375, -0.2509765625, 0.341064453125, -0.88818359375, -0.955078125, -0.447998046875, 1.046875, -0.2410888671875, 0.75341796875, -0.2071533203125, -0.94921875, -0.381103515625, -0.8935546875, 0.0155792236328125, -0.8896484375, 0.319580078125, -0.43310546875, 1.0771484375, -0.45068359375, 0.356201171875, 0.3798828125, 0.458740234375, -0.9248046875, 0.2900390625, -1.3955078125, 0.79541015625, -1.1328125, -0.72509765625, -1.71484375, -1.0966796875, 0.7021484375, -0.0240325927734375, -0.97314453125, 0.73388671875, 0.4609375, 0.2066650390625, 0.492919921875, -0.035980224609375, 0.185302734375, -1.2265625, -0.06951904296875, -0.1190185546875, 2.5, -1.80859375, -0.0204010009765625, 0.48046875, -0.2476806640625, -0.11767578125, 0.462158203125, 0.90185546875, 0.568359375, 1.140625, 1.0224609375, 0.3349609375, -0.2490234375, -1.0205078125, -0.81982421875, 1.0947265625, -0.72021484375, 0.453125, -0.5673828125, -1.3916015625, -0.76416015625, 0.5087890625, 0.71044921875, 0.57861328125, -0.73291015625, 0.449951171875, -2.4296875, -0.67236328125, -0.36279296875, 0.254150390625, 1.6240234375, -0.8134765625, 0.6123046875, 1.853515625, -1.4033203125, 0.6552734375, 0.6279296875, -0.010650634765625, -1.32421875, 0.2421875, 0.78759765625, 0.440185546875, -0.6533203125, 0.2340087890625, -1.2509765625, -0.135498046875, 0.61767578125, -0.78271484375, -1.267578125, -0.2529296875, 1.111328125, -1.03125, -1.291015625, -1.1474609375, -0.407958984375, -0.71142578125, 1.052734375, 0.935546875, -2.4140625, 1.1787109375, 0.01318359375, -1.537109375, -0.59912109375, -0.70654296875, 0.268310546875, -0.004306793212890625, 0.7275390625, -0.042938232421875, -0.85888671875, -0.25732421875, 1.2099609375, -1.9169921875, 0.58642578125, -1.1396484375, -0.09808349609375, 0.61474609375, -0.252685546875, 0.10162353515625, 0.83056640625, 0.181640625, -0.170166015625, 1.095703125, 0.27392578125, 1.7607421875, 2.076171875, 1.0185546875, -0.93603515625, -0.13623046875, -0.254150390625, -1.4912109375, 0.332763671875, 1.35546875, -1.072265625, 0.0236663818359375, 1.353515625, 1.5224609375, 0.216796875, -0.1552734375, -0.8408203125, 0.12744140625, 0.4990234375, 0.2412109375, -0.59326171875, -0.81689453125, 1.2783203125, 0.302978515625, 0.17626953125, -0.55859375, 1.05859375, 1.1962890625, 0.4892578125, 1.1357421875, -0.50537109375, -1.5205078125, 0.96044921875, 1.7509765625, 0.86865234375, 1.1416015625, 0.80517578125, 0.79248046875, -0.59130859375, 2.12109375, 1.93359375, -0.5654296875, -1.791015625, 0.5302734375, 0.7578125, -0.8662109375, 0.33251953125, -0.64892578125, -0.78662109375, 0.300048828125, -0.2415771484375, -0.82177734375, 1.3251953125, -0.8115234375, 1.8974609375, -1.0341796875, 0.92138671875, 1.224609375, -1.091796875, 0.86376953125, -0.349853515625, -0.06549072265625, -0.58837890625, -0.133544921875, 0.6767578125, -0.29833984375, -1.138671875, 1.4287109375, 1.2998046875, -0.6015625, 1.5751953125, -1.6640625, 1.5673828125, 0.26953125, -0.260498046875, -0.71435546875, -0.4814453125, -1.337890625, 1.0400390625, 0.264404296875, -0.31689453125, -1.9306640625, -0.07550048828125, -0.1270751953125, 0.7958984375, -1.568359375, -0.2744140625, 0.236572265625, 0.8310546875, -0.0178070068359375, -1.0126953125, 1.490234375, -1.599609375, -0.09991455078125, -0.41845703125, 0.182861328125, -0.0794677734375, 0.434814453125, 0.3955078125, -1.2412109375, 1.275390625, -1.6689453125, 0.10870361328125, 1.01953125, -1.365234375, 0.393310546875, 1.201171875, 0.4697265625, -1.1884765625, 1.072265625, -0.1812744140625, 0.177490234375, 0.11895751953125, 0.230224609375, -2.55078125, 0.34765625, -0.66845703125, 0.302490234375, 0.97314453125, 0.69140625, -0.7783203125, 0.8876953125, 1.078125, 0.38232421875, 0.1505126953125, 0.05615234375, 1.5673828125, 0.74609375, 0.8896484375, 0.263427734375, -0.44775390625, -0.56884765625, 0.166015625, -0.8876953125, -0.72705078125, 1.0693359375, -0.31494140625, -1.068359375, -1.203125, 1.6943359375, 0.02685546875, 1.65234375, 0.288818359375, 0.52490234375, 0.5, -2.095703125, -0.204833984375, -0.043548583984375, 0.689453125, 2.125, -0.1475830078125, 0.63525390625, 1.4619140625, -0.8740234375, 1.001953125, 0.81298828125, -0.2010498046875, 1.0751953125, -1.458984375, -0.95361328125, -2.66015625, 1.255859375, 0.3857421875, 1.9453125, -0.00724029541015625, -3.375, 0.53076171875, 0.484375, 1.1171875, -0.5615234375, -1.2373046875, -0.478759765625, -0.68994140625, 1.9658203125, -0.231201171875, 0.0006604194641113281, 0.67041015625, 0.89892578125, 0.89501953125, -0.336181640625, -1.3720703125, -0.92578125, -2.09765625, -1.3896484375, -1.314453125, -0.2337646484375, 0.186279296875, -0.285400390625, -1.0849609375, -0.287109375, 0.049224853515625, 0.231689453125, 0.44091796875, 0.64306640625, 0.43505859375, -1.728515625, -1.1650390625, 1.09375, -0.58837890625, 0.488037109375, 0.6123046875, 0.42724609375, -0.6904296875, -0.11328125, -1.4912109375, -0.29931640625, -2.125, -1.634765625, -0.88720703125, 0.98388671875, -2.01171875, 0.61962890625, -1.287109375, 1.56640625, -2.599609375, 0.65625, -0.970703125, 1.23046875, -1.1357421875, 0.06719970703125, -0.54638671875, 1.03125, 1.9716796875, 0.78125, 0.5517578125, 0.0390625, 0.433837890625, -0.481689453125, 1.529296875, -1.658203125, -2.046875, -0.1475830078125, -0.998046875, 0.3515625, 0.30126953125, 1.4150390625, -0.26416015625, 0.2213134765625, 0.6025390625, -1.3515625, 0.461669921875, -0.15576171875, -0.425048828125, -0.5830078125, -1.6630859375, -1.4130859375, 0.1173095703125, 0.51953125, 0.9931640625, -0.330078125, 1.1953125, 0.10211181640625, 2.037109375, -1.2685546875, -0.529296875, 0.482177734375, 0.8212890625, -0.332275390625, -0.48291015625, -0.603515625, 0.19091796875, 0.77783203125, 1.546875, 0.0657958984375, 0.81591796875, 0.10211181640625, 0.2340087890625, 1.6728515625, -1.25390625, 0.322998046875, -0.72998046875, -0.0112762451171875, 1.2861328125, 0.38916015625, -1.021484375, -0.62060546875, -0.182861328125, -0.60791015625, -0.0369873046875, -1.15234375, -0.235595703125, 0.85888671875, -0.3466796875, -0.4765625, -0.8271484375, 2.48828125, 0.0833740234375, 0.390625, 1.330078125, -0.2196044921875, 0.7158203125, 0.07232666015625, -0.85498046875, 0.93017578125, 0.88134765625, -0.404541015625, -0.345458984375, 0.74267578125, 0.76904296875, -0.76708984375, 0.384033203125, 0.3466796875, -0.374755859375, -0.630859375, -0.445068359375, -1.0771484375, -0.049346923828125, 1.2451171875, 0.462890625, -2.318359375, 1.04296875, 0.99755859375, 1.712890625, 1.35546875, 0.5537109375, 1.103515625, -0.642578125, 0.12164306640625, -0.1822509765625, 0.5126953125, -2.529296875, 0.06787109375, -0.783203125, -0.10247802734375, -0.55517578125, -0.423583984375, 0.7529296875, 0.393310546875, 0.42578125, 0.2418212890625, -0.7236328125, 1.5419921875, 0.8232421875, 0.006500244140625, -0.359375, -0.64990234375, -0.900390625, -0.01377105712890625, 1.1181640625, 1.3515625, -1.330078125, -0.82421875, 0.01094818115234375, 1.1572265625, -1.5615234375, -0.487060546875, 0.3564453125, 1.2587890625, 1.32421875, 0.2119140625, 1.30859375, -0.75927734375, -1.9951171875, -1.509765625, -0.1903076171875, 1.689453125, 0.8095703125, 0.86474609375, -0.27685546875, 0.292724609375, 0.205078125, 1.0185546875, 1.478515625, 1.4833984375, 0.41455078125, -0.46240234375, 0.09625244140625, -1.6689453125, 0.1741943359375, -0.48046875, -0.75537109375, -0.274658203125, 2.01953125, 0.2498779296875, -0.2022705078125, -1.7236328125, 0.19921875, -0.036956787109375, -0.7255859375, 0.19921875, -0.12255859375, 0.7138671875, -0.75146484375, -0.2198486328125, -0.09503173828125, 0.42138671875, 0.0673828125, 1.5751953125, 1.37890625, 0.625, 0.45263671875, 0.409912109375, -1.0712890625, -0.304443359375, 2.6015625, 0.90576171875, -1.2412109375, -0.23388671875, -1.177734375, 0.716796875, 0.60302734375, -1.029296875, -0.052764892578125, -0.417724609375, 1.0947265625, 1.521484375, 0.6318359375, 0.271728515625, 1.0146484375, -0.8603515625, -0.435302734375, 0.373046875, -1.0146484375, 0.2017822265625, 1.7763671875, -0.2161865234375, 1.4658203125, -1.3544921875, 1.2255859375, -2.427734375, -0.35107421875, 0.415283203125, -0.95751953125, 0.197265625, -0.80908203125, -1.0048828125, 1.1552734375, -1.236328125, 0.0396728515625, -0.978515625, 1.904296875, 0.390380859375, 0.6474609375, -1.6552734375, -1.8623046875, 2.0, -0.92041015625, -0.224853515625, -0.365478515625, -0.143310546875, 0.244384765625, -0.99658203125, 0.328369140625, -0.28125, 1.173828125, -1.0556640625, -1.1474609375, 0.732421875, 0.2442626953125, -0.59521484375, 0.51611328125, 0.2237548828125, 1.1474609375, -0.205810546875, -0.27978515625, 0.52783203125, -0.49951171875, 0.35498046875, 1.6865234375, -0.705078125, -1.9482421875, 0.08746337890625, -1.7919921875, -0.0012102127075195312, -0.86572265625, -0.69775390625, -0.2978515625, -1.1259765625, -1.5830078125, 0.97412109375, -0.03912353515625, 0.487548828125, -1.8701171875, 0.87890625, 1.5859375, 0.1676025390625, 0.426513671875, 1.25, 0.939453125, -0.9423828125, -2.158203125, 1.12890625, -1.4912109375, -0.41162109375, -1.0517578125, -1.4775390625, -0.81005859375, -0.5458984375, 0.42431640625, -0.83837890625, 0.42236328125, -0.51123046875, 0.1429443359375, -1.7021484375, 0.55908203125, -1.439453125, -0.57470703125, 1.2587890625, 0.48876953125, -0.3291015625, 1.0751953125, 1.2939453125, -0.173583984375, 0.128662109375, -0.5947265625, -0.53466796875, -0.16845703125, 0.247314453125, -0.43408203125, -0.79052734375, 0.46435546875, 1.0029296875, 1.1376953125, 1.0615234375, -0.48486328125, 1.6552734375, -0.0654296875, 0.55810546875, -0.297607421875, -0.67236328125, -1.1376953125, 0.75390625, 0.441650390625, -0.54931640625, -0.52001953125, -2.634765625, 1.1123046875, 0.66357421875, -0.11138916015625, 0.51123046875, 2.314453125, -1.4619140625, -0.2298583984375, 0.75537109375, 1.8505859375, 0.93115234375, -1.123046875, -0.49853515625, -0.2130126953125, 0.313720703125, -0.79541015625, -0.8525390625, -1.3310546875, 0.102294921875, -0.384521484375, -0.273681640625, 0.1385498046875, 0.732421875, 0.20166015625, -0.2059326171875, 0.76513671875, -0.74072265625, 0.158203125, -0.029205322265625, -2.080078125, 0.48779296875, 0.1875, -1.13671875, -2.07421875, 1.1015625, -0.732421875, -1.830078125, -1.09765625, -2.275390625, -0.71435546875, -0.76416015625, 1.71484375, -0.57958984375, -0.59521484375, 0.044921875, 0.0635986328125, -1.9658203125, 0.888671875, -1.5703125, -0.373779296875, 0.7939453125, -0.72265625, -1.90234375, 0.283935546875, 0.65869140625, -0.08575439453125, 0.235107421875, -0.2352294921875, 1.2080078125, 1.380859375, 1.10546875, 0.054046630859375, 0.78076171875, 1.37890625, -0.73876953125, 0.12127685546875, -0.007083892822265625, -0.40673828125, -1.5029296875, 0.56396484375, -0.9677734375, -0.025787353515625, -1.02734375, 0.72705078125, -0.767578125, 0.328369140625, -0.63818359375, -0.5478515625, -0.09503173828125, 0.46875, 0.51953125, 0.6318359375, -0.58203125, -0.71484375, 1.029296875, -0.033355712890625, 0.8896484375, -0.375244140625, 0.291015625, -1.55078125, -0.81640625, -0.473388671875, 0.5205078125, 0.439208984375, 1.1708984375, -0.6494140625, 0.7724609375, -0.460693359375, -0.09161376953125, 0.89306640625, 0.974609375, 1.419921875, 1.4072265625, 0.302490234375, -1.3974609375, -0.7294921875, 2.041015625, -0.55419921875, -1.794921875, 0.1943359375, 1.2841796875, 0.481201171875, 0.2178955078125, 0.5869140625, -1.0380859375, -0.34423828125, -1.7265625, -0.64892578125, -0.345703125, -0.42236328125, 1.1123046875, 1.6298828125, 0.07275390625, 0.421875, 1.5341796875, 0.759765625, -1.0078125, 0.875, -0.274169921875, 1.3818359375, -0.299560546875, -0.168212890625, 0.8759765625, -1.1044921875, -0.599609375, -0.1558837890625, 1.6416015625, 1.1318359375, 0.151611328125, 0.6416015625, 0.1556396484375, 0.66796875, 1.021484375, 1.8046875, 0.341064453125, 0.38916015625, -0.890625, -0.489501953125, 0.6064453125, 1.75, 1.1416015625, -0.370361328125, -0.2109375, 0.2149658203125, 0.84521484375, -1.6572265625, 1.59765625, -0.17236328125, 2.185546875, 0.9541015625, 0.05267333984375, -1.4541015625, -1.0068359375, 0.8955078125, -0.0906982421875, -0.7177734375, -1.3330078125, 1.203125, 0.4404296875, 0.2763671875, 0.11810302734375, -2.607421875, -0.79833984375, -1.08984375, 0.38818359375, 0.407470703125, 0.18359375, 0.3291015625, -0.85986328125, 0.4921875, 0.373291015625, -0.77099609375, -0.94287109375, -1.720703125, 0.55859375, -1.2958984375, 1.0654296875, 1.921875, 1.46484375, -0.5517578125, -0.2386474609375, -1.4912109375, -1.712890625, -1.8056640625, -0.60595703125, 0.64501953125, 0.1788330078125, 1.20703125, -0.52294921875, -0.188232421875, -0.2176513671875, 2.203125, -1.861328125, -0.11474609375, -0.56005859375, 0.2138671875, -0.6640625, -1.962890625, 0.07110595703125, 2.744140625, -0.384521484375, -0.3974609375, -1.5517578125, -0.58740234375, -0.4716796875, -1.82421875, 0.159912109375, 0.90478515625, -0.3076171875, -2.447265625, -0.66552734375, -2.01171875, 0.1761474609375, 0.57958984375, -0.488525390625, 0.183349609375, -0.20263671875, -0.3603515625, -0.68994140625, -1.5283203125, -0.1328125, 0.72900390625, 0.525390625, 0.654296875, 1.97265625, -1.220703125, -0.51171875, 2.2265625, 0.6044921875, 0.9658203125, -1.328125, 0.11834716796875, -0.0662841796875, -0.43896484375, -0.82470703125, 3.283203125, 0.043365478515625, -0.01324462890625, 0.40771484375, -1.76171875, -0.293212890625, 0.7236328125, -1.9501953125, 0.5322265625, 0.243896484375, -1.5615234375, -0.1783447265625, 0.6083984375, -0.086669921875, -1.3994140625, 0.89013671875, -0.8388671875, 0.57568359375, 0.12310791015625, -1.4013671875, 1.3212890625, 0.86669921875, 0.407470703125, -0.270751953125, 0.264892578125, 0.2147216796875, -0.1361083984375, 0.1114501953125, -0.6884765625, 0.213623046875, -1.0234375, 0.78857421875, 0.177490234375, -0.458740234375, -1.970703125, 0.04571533203125, -0.10638427734375, 0.431884765625, -1.25, -0.1759033203125, 1.5673828125, -0.70703125, 0.9267578125, -1.8408203125, -2.06640625, -0.1300048828125, 1.30078125, -0.136474609375, -0.56396484375, 1.0693359375, 1.228515625, 1.4443359375, -1.77734375, 1.359375, -1.1259765625, -0.183349609375, 0.042633056640625, -0.44921875, -0.1585693359375, -1.3056640625, 1.2021484375, -0.37451171875, 1.46484375, 1.0185546875, 0.693359375, 0.68017578125, -0.0916748046875, -1.4853515625, -0.41064453125, -0.7041015625, -0.6376953125, 0.91845703125, 0.5048828125, -1.59765625, 0.3701171875, -1.5478515625, 0.412109375, 0.326171875, -0.13330078125, 1.595703125, -1.09765625, -0.482666015625, -0.2158203125, -0.354736328125, -0.3515625, -0.29541015625, 2.07421875, 0.63623046875, -0.384033203125, 0.256591796875, -0.469970703125, -0.207275390625, -0.6904296875, 0.84130859375, 1.9404296875, 0.489501953125, -1.2138671875, 0.92626953125, 0.79736328125, 1.5517578125, 1.1005859375, -1.25, -1.4404296875, -2.4296875, -0.9814453125, -0.2076416015625, -0.475830078125, 0.327880859375, 1.9345703125, -0.348388671875, 0.1014404296875, 1.611328125, 1.1845703125, 0.96923828125, -2.94140625, -0.488525390625, 0.38818359375, -0.34326171875, -0.609375, 0.445068359375, -2.3046875, 1.6142578125, -1.1015625, -1.177734375, 0.19384765625, 0.36376953125, -0.439453125, 0.73095703125, 1.05078125, -0.224609375, 2.23828125, -0.66748046875, 1.40234375, 0.248779296875, -1.6923828125, 0.36181640625, 1.826171875, -0.5419921875, 0.140380859375, 0.8681640625, -0.6337890625, 0.49169921875, 1.1611328125, 0.85302734375, 0.833984375, -1.9189453125, 1.0107421875, 0.00861358642578125, -0.419677734375, 1.099609375, -0.1802978515625, -0.54345703125, 1.1513671875, 0.2056884765625, -1.3447265625, 0.146240234375, -0.8994140625, -0.5771484375, 0.8720703125, -0.67578125, 0.1973876953125, -0.0237274169921875, -0.158203125, -0.0390625, 0.83740234375, 0.7939453125, -0.034759521484375, -1.2861328125, 0.6748046875, -0.275634765625, -1.1533203125, 0.2474365234375, -0.40087890625, -0.9873046875, 0.1478271484375, 0.83837890625, -1.1953125, -0.892578125, 1.205078125, 0.94775390625, 0.347900390625, 1.8896484375, 0.1817626953125, -0.5712890625, -1.1162109375, -0.60986328125, 0.03564453125, 0.0038814544677734375, -1.01171875, -0.982421875, -0.0941162109375, 0.032379150390625, -1.01953125, -0.33935546875, -2.20703125, 1.34765625, 2.111328125, -0.48388671875, -0.60888671875, 2.294921875, 2.248046875, -0.31201171875, 0.68359375, -1.775390625, 0.787109375, -0.403076171875, 0.85546875, -0.2159423828125, -0.1783447265625, 0.56005859375, -0.548828125, -1.4453125, 0.480224609375, 1.625, 1.2626953125, 0.050506591796875, -0.0391845703125, 0.68017578125, -0.958984375, 1.34765625, 0.239501953125, -1.1826171875, 0.24853515625, 0.05413818359375, -0.372314453125, -0.424560546875, -1.0205078125, 0.333251953125, 0.75927734375, -0.299072265625, -0.8837890625, 0.619140625, 0.35205078125, 1.865234375, -0.77490234375, 0.99462890625, 0.12158203125, 0.148681640625, -0.54736328125, 0.405517578125, -0.95263671875, 0.6845703125, 0.168212890625, 0.104736328125, 0.320068359375, -1.2470703125, 0.33984375, 0.313720703125, 0.049163818359375, -0.60791015625, 1.578125, -1.259765625, -0.228271484375, 0.4931640625, 0.4189453125, 0.4140625, -0.7314453125, -1.9580078125, -0.005550384521484375, -0.1719970703125, -0.462890625, -0.1658935546875, 1.537109375, 0.5849609375, 1.4208984375, -0.380615234375, 0.81640625, 0.408935546875, -0.444580078125, 0.392578125, 0.061614990234375, 1.8876953125, 0.5478515625, 0.2763671875, -0.2193603515625, -1.0244140625, -1.23046875, -0.90771484375, -0.53173828125, -0.65478515625, -2.58203125, 0.66064453125, 2.53515625, 0.062255859375, -0.9873046875, 0.1304931640625, -1.3876953125, -0.5390625, -0.53515625, -0.445068359375, -0.1397705078125, 0.36865234375, 0.2158203125, 0.0919189453125, -0.11260986328125, -0.315185546875, -0.320556640625, -0.399169921875, -0.4013671875, -1.328125, -0.21826171875, 0.273681640625, -0.75927734375, -0.931640625, 1.119140625, 0.81103515625, -2.24609375, -0.0413818359375, 1.482421875, 0.240478515625, -0.033416748046875, 0.60498046875, 1.1064453125, -1.515625, -1.15234375, -1.771484375, 2.259765625, -0.476806640625, -0.390380859375, 0.092041015625, -0.8359375, 0.418701171875, -1.0380859375, -0.9833984375, 1.36328125, -0.384765625, -0.4111328125, 0.41650390625, 0.347900390625, -0.1856689453125, -1.689453125, 0.86279296875, -1.6474609375, -0.1573486328125, -1.7587890625, -1.1064453125, -1.49609375, 0.58935546875, -0.27392578125, -0.337646484375, -0.81005859375, -0.494140625, 0.32958984375, 0.441162109375, 1.0634765625, 0.08306884765625, -0.1729736328125, 0.72509765625, -0.1468505859375, -1.6865234375, -0.86328125, -0.91748046875, -1.3193359375, -0.018096923828125, 2.267578125, -1.1171875, -0.19482421875, 1.048828125, -0.411376953125, 0.495361328125, -1.859375, 0.1038818359375, -1.0546875, 1.0556640625, -0.427001953125, 0.67919921875, 0.2193603515625, 1.4150390625, -0.2158203125, -1.2158203125, 0.50341796875, -0.1788330078125, -0.04974365234375, 0.40625, 1.3896484375, 0.495849609375, -1.1787109375, 0.362060546875, -0.53759765625, 0.86328125, 0.30908203125, -0.1226806640625, -0.779296875, -0.33642578125, -1.203125, 0.55126953125, 1.3486328125, -0.5673828125, -2.08984375, 0.93603515625, -0.448486328125, 1.0869140625, 1.69921875, -0.111083984375, 0.98681640625, 0.46875, -0.104736328125, 0.078369140625, 1.203125, 0.1278076171875, 1.677734375, 0.237060546875, -0.487548828125, -0.51953125, -0.0867919921875, 0.186767578125, -0.02459716796875, -0.5693359375, -2.55078125, 0.030364990234375, -0.92822265625, -0.0653076171875, -0.873046875, 0.7001953125, 0.411376953125, -2.125, -0.356201171875, 0.7373046875, -0.31591796875, -0.07244873046875, 0.1680908203125, 0.55419921875, -0.42431640625, -0.65283203125, 1.212890625, 0.1668701171875, 0.47802734375, 0.51708984375, 0.99169921875, 0.79931640625, 0.51416015625, -0.88525390625, -0.25341796875, 0.2744140625, 1.107421875, 1.1376953125, 0.31884765625, -0.354248046875, -1.1005859375, 1.8984375, -0.2020263671875, -0.272216796875, -0.1485595703125, -0.050567626953125, -2.73828125, 1.4892578125, -0.311767578125, -0.09844970703125, 0.83447265625, -0.0799560546875, -0.58544921875, 0.260009765625, 0.69677734375, -0.66796875, -0.0029754638671875, 0.313720703125, 0.54443359375, -0.1790771484375, -1.3427734375, 1.4013671875, 0.285888671875, 1.2099609375, -1.501953125, 0.61279296875, 1.25390625, -0.76025390625, 0.94921875, -0.392822265625, -1.0, -0.05712890625, -0.646484375, -0.5029296875, -2.318359375, -1.4951171875, 0.64306640625, -0.19140625, -1.298828125, 0.29443359375, -0.732421875, 0.892578125, 0.84716796875, 0.90380859375, -0.11907958984375, -1.3896484375, -1.1591796875, -0.26220703125, 1.439453125, 1.1865234375, -1.443359375, 0.2371826171875, -0.52587890625, 1.87109375, 0.7021484375, 0.2763671875, -0.830078125, 0.642578125, -0.79541015625, -0.0833740234375, -1.11328125, 0.468017578125, -0.356689453125, 1.70703125, 0.9130859375, 0.27734375, -1.8642578125, 1.2890625, -1.443359375, -0.206787109375, 0.092529296875, 1.34765625, 0.036407470703125, -0.07025146484375, 0.2188720703125, -1.486328125, -0.1051025390625, 1.662109375, 0.8466796875, -0.401123046875, 0.50634765625, 0.4482421875, -1.3056640625, 0.068603515625, 0.448486328125, -0.3681640625, 0.90478515625, -0.0006556510925292969, 1.0810546875, 1.4013671875, -0.8193359375, -0.7080078125, 0.7978515625, 0.152099609375, -0.00234222412109375, 1.3662109375, -0.4287109375, -0.02044677734375, 1.267578125, 1.3046875, -1.1474609375, 0.92431640625, -0.43212890625, 0.63330078125, 0.1461181640625, 1.0107421875, -1.1484375, -0.1448974609375, 0.669921875, -0.46240234375, -0.494873046875, -0.31494140625, -0.53662109375, -1.4755859375, 0.3818359375, -1.5380859375, -0.89794921875, 0.293212890625, 1.7587890625, 0.27978515625, -0.95703125, 0.3369140625, 0.1822509765625, -0.44775390625, -0.89697265625, 0.1585693359375, -0.81494140625, 0.54541015625, 1.330078125, 0.55224609375, -0.80029296875, -0.7607421875, -0.049591064453125, 0.810546875, -0.7666015625, 0.775390625, -1.7236328125, 0.041534423828125, -0.2088623046875, -0.99462890625, 1.5234375, 0.5458984375, 0.274658203125, -0.351318359375, -0.30859375, -0.3203125, -0.95703125, 0.381591796875, 1.6640625, 0.8984375, 0.40185546875, -0.5244140625, -1.9248046875, -1.1875, 0.396240234375, -0.6787109375, -0.6650390625, -1.005859375, 0.473388671875, -0.1947021484375, -0.880859375, -0.14697265625, 0.1573486328125, -1.2275390625, 1.7021484375, -0.3564453125, -0.11859130859375, -1.7919921875, 0.79833984375, -0.2939453125, -0.94677734375, -0.392822265625, -0.2293701171875, 0.6669921875, 0.5087890625, 0.37841796875, -0.345703125, -0.125, -0.18798828125, 2.65625, -1.93359375, 0.78369140625, -1.2802734375, 1.5537109375, -0.7138671875, -0.55078125, 0.3876953125, -0.6806640625, 0.414794921875, -0.90576171875, -0.62841796875, -0.673828125, 0.040924072265625, 0.6884765625, 0.89111328125, -1.0673828125, 1.0810546875, -1.8505859375, 0.052001953125, 0.5888671875, -0.389892578125, 0.45068359375, -1.9404296875, -1.1298828125, -2.205078125, -0.088623046875, 0.218017578125, 1.2275390625, -0.5361328125, 1.4990234375, 1.0546875, -0.93017578125, 0.1690673828125, 0.007610321044921875, 0.84033203125, 0.43994140625, 0.440185546875, -0.92578125, -0.7138671875, -0.2939453125, -1.38671875, -0.81640625, 0.8134765625, -0.289794921875, -0.2191162109375, -0.5703125, -0.394775390625, -0.7119140625, -1.9560546875, 0.462646484375, 0.241943359375, 0.4287109375, -1.2626953125, -1.818359375, -0.1832275390625, -0.94140625, -0.36328125, 0.330078125, 0.55908203125, 0.89794921875, 0.8583984375, -1.3193359375, 0.505859375, -0.2095947265625, -1.35546875, -1.486328125, 0.77099609375, -2.060546875, -1.6181640625, -1.3564453125, 0.837890625, 1.39453125, -0.179931640625, 0.71435546875, 1.4853515625, 1.078125, 0.580078125, 0.17919921875, -1.1904296875, -0.181396484375, 0.09075927734375, -2.55078125, 2.435546875, 0.70849609375, 0.5791015625, -0.27099609375, 0.482421875, 0.64794921875, -1.20703125, -0.92822265625, 0.568359375, 0.60400390625, -1.60546875, -0.25927734375, 0.86865234375, -1.0068359375, 0.159423828125, 0.5576171875, -0.87158203125, -0.4404296875, -1.728515625, -0.35791015625, -0.74755859375, -0.280517578125, -2.123046875, 0.70703125, -1.75, -0.252685546875, -0.251220703125, -0.6806640625, -0.7763671875, -1.2939453125, 0.64794921875, -0.697265625, 0.36083984375, -0.65966796875, 0.375732421875, -0.279052734375, 0.1337890625, -1.5185546875, 1.10546875, 1.74609375, -1.71484375, -1.029296875, -0.76708984375, 0.236572265625, -0.032867431640625, -0.427490234375, 1.1083984375, 0.377197265625, -0.63134765625, -0.2432861328125, 0.26953125, -1.5849609375, -0.016815185546875, -0.070068359375, -0.102294921875, 1.7666015625, -1.2978515625, -1.541015625, 0.8974609375, 1.2421875, 0.1663818359375, 0.6787109375, -1.935546875, 0.343505859375, -1.134765625, -0.54248046875, -0.212890625, -0.160400390625, 0.418212890625, 0.3466796875, 0.84716796875, -0.2108154296875, -0.316650390625, 0.78515625, -1.326171875, 0.0169525146484375, 1.4130859375, 0.30517578125, -0.036224365234375, 1.7421875, -0.0706787109375, 0.701171875, -1.025390625, -1.8330078125, -0.0518798828125, 1.1015625, 0.80126953125, -0.33203125, 0.3349609375, 0.8134765625, 1.962890625, -0.6220703125, 0.5810546875, 0.84375, 0.155029296875, -0.75, 0.3701171875, -1.0302734375, -1.552734375, 1.41796875, 0.0019435882568359375, -0.0019893646240234375, -1.8759765625, -1.1181640625, -0.38232421875, 1.40625, 0.70068359375, -0.96923828125, -0.54833984375, -0.484619140625, 0.441650390625, 0.0899658203125, -1.5146484375, 0.22607421875, -0.333740234375, 1.3017578125, 0.037322998046875, 0.68994140625, 1.107421875, -0.08404541015625, -0.486328125, -2.912109375, -1.841796875, -1.421875, -0.493408203125, -1.767578125, 0.07958984375, 0.44921875, 1.357421875, 0.01239013671875, 1.23828125, 0.28759765625, -0.359130859375, 0.91455078125, -0.75048828125, 0.3330078125, 1.046875, 1.63671875, 0.62548828125, 0.283203125, -0.296875, -0.23681640625, 0.36865234375, -0.16357421875, -1.7333984375, -0.6611328125, -1.177734375, -0.5810546875, -1.3642578125, 0.267578125, -1.2802734375, 0.81103515625, -0.489501953125, -0.44091796875, 3.427734375, 0.576171875, 0.1343994140625, 1.919921875, -0.7490234375, 1.400390625, -0.412841796875, 1.3505859375, 0.60693359375, -1.2900390625, -0.446044921875, 0.061859130859375, 1.5986328125, -1.2958984375, -1.357421875, -1.857421875, 0.2100830078125, -0.6513671875, 0.414306640625, 0.7021484375, -0.1353759765625, -0.4892578125, -0.183837890625, -1.2890625, -0.73046875, -2.033203125, -0.45166015625, -0.09979248046875, 0.0244140625, 0.98681640625, 0.92236328125, 0.919921875, -0.884765625, 0.470947265625, -0.51318359375, 0.6611328125, 0.1456298828125, 0.16064453125, -0.195068359375, -1.345703125, -1.01171875, 0.861328125, 0.1787109375, 0.45849609375, -0.176513671875, 0.284423828125, 0.9580078125, -0.310302734375, -1.201171875, -0.6328125, -1.1435546875, 0.50927734375, -0.07220458984375, -0.0261383056640625, 0.556640625, -0.0460205078125, 1.361328125, 1.45703125, -1.1201171875, 1.40234375, -0.3720703125, -0.02874755859375, 1.064453125, -0.2083740234375, -0.303466796875, -0.282470703125, -0.53662109375, -0.0223846435546875, 0.00685882568359375, -0.388671875, 0.4580078125, -0.4306640625, -0.0221405029296875, -0.9013671875, 0.325927734375, 0.1693115234375, 0.180908203125, -0.317138671875, 0.4248046875, -0.063232421875, 0.4736328125, 1.5791015625, 0.11572265625, -0.60302734375, -0.4638671875, -1.50390625, -1.1162109375, 1.3984375, 0.1104736328125, 0.09674072265625, 0.81640625, 1.1796875, 0.1943359375, 1.140625, -0.004215240478515625, -1.3212890625, 1.0908203125, -0.1650390625, -0.1385498046875, -0.533203125, 0.53955078125, 1.1015625, -0.6103515625, -0.0391845703125, -0.366455078125, 0.69287109375, -0.7421875, 0.12841796875, -1.87890625, 0.119384765625, 1.0205078125, -0.08990478515625, 0.64306640625, -1.3662109375, -0.18994140625, -1.0146484375, 0.12359619140625, -1.828125, 0.2235107421875, -0.485107421875, 1.0849609375, -1.2060546875, 1.64453125, -0.458984375, -1.3408203125, -0.08209228515625, -0.035736083984375, -0.81298828125, 1.048828125, -0.50439453125, 1.68359375, -0.0853271484375, -0.1324462890625, -0.10205078125, -2.208984375, -1.375, -1.142578125, 2.044921875, 0.53515625, -0.69189453125, 0.349609375, -0.43115234375, -0.86962890625, 0.65771484375, -0.57080078125, -0.69189453125, 1.2392578125, 0.75439453125, 1.5537109375, -0.482177734375, 0.51171875, -1.486328125, 0.044097900390625, 0.18017578125, 0.211181640625, 0.052825927734375, -0.166748046875, 0.79443359375, -1.986328125, -1.6455078125, -0.1317138671875, -0.380126953125, -2.296875, -0.2122802734375, -0.69287109375, -0.85400390625, 0.0025577545166015625, -0.47802734375, 0.009735107421875, -0.90087890625, 0.332275390625, -0.9404296875, -1.126953125, 1.443359375, 1.21484375, 1.01171875, 0.319580078125, 0.434326171875, 0.281494140625, 0.67724609375, 0.33154296875, -0.5302734375, -1.0771484375, -0.67431640625, -0.51171875, 0.0310516357421875, 0.326171875, -0.904296875, 0.340576171875, -1.8349609375, -1.25390625, 1.8486328125, -0.07794189453125, 0.79296875, 0.140625, -1.5498046875, -1.0615234375, 0.41796875, 0.2239990234375, -0.42822265625, 0.7216796875, 2.625, 1.23046875, 0.6064453125, 2.029296875, 0.82861328125, -0.26513671875, -1.6083984375, 0.57568359375, -0.1640625, -0.5419921875, 0.9599609375, -0.263916015625, 1.0302734375, 1.365234375, -0.338134765625, 0.1341552734375, 0.87451171875, -1.0361328125, -1.2392578125, -1.3291015625, -2.98046875, 0.99169921875, 0.79443359375, -0.058502197265625, 0.2305908203125, -1.7734375, 2.72265625, 1.2021484375, -0.122802734375, -0.509765625, 0.493408203125, 0.367431640625, -0.78369140625, -0.34716796875, -1.033203125, -1.1337890625, -0.10174560546875, 0.2685546875, 0.451171875, -0.568359375, 0.98193359375, -0.322021484375, 2.001953125, 0.685546875, -0.1119384765625, -1.0751953125, -2.08203125, 0.1595458984375, -0.70654296875, -0.740234375, -0.311767578125, 0.6396484375, -0.386474609375, -1.765625, -1.1474609375, 0.81591796875, 1.2001953125, 1.1123046875, 1.203125, 0.0235748291015625, 0.239990234375, -0.91943359375, 0.9013671875, 0.018463134765625, 1.072265625, -1.33984375, 0.939453125, 1.05078125, -0.06011962890625, 0.88623046875, -0.5146484375, 0.71826171875, 0.66455078125, 1.35546875, 2.18359375, -0.86865234375, -0.51123046875, -2.1953125, -0.271240234375, -0.0706787109375, 1.181640625, 0.2078857421875, 0.1119384765625, 2.400390625, -1.685546875, -0.50341796875, -1.1552734375, 0.409912109375, 0.73681640625, -0.2509765625, -1.2060546875, -0.8251953125, 1.32421875, 0.791015625, 0.783203125, -0.728515625, -0.861328125, -0.480224609375, -0.18115234375, 0.701171875, 0.6279296875, -0.07781982421875, 0.55712890625, 0.3134765625, -0.72998046875, -0.1700439453125, 0.197509765625, 0.35107421875, 1.4287109375, -1.8369140625, -0.39404296875, -0.33544921875, 0.2279052734375, 0.8671875, 0.489501953125, 0.763671875, 0.05755615234375, -0.07427978515625, 0.693359375, 0.96826171875, -1.203125, 0.08697509765625, -0.4140625, -0.5224609375, -0.5869140625, 0.359130859375, -2.048828125, -0.2003173828125, -1.2275390625, -0.07012939453125, -2.216796875, 1.9609375, -1.333984375, 0.548828125, -0.56298828125, -1.0810546875, 0.1558837890625, 0.188232421875, 1.5283203125, -0.448486328125, 0.2362060546875, -0.720703125, 1.1533203125, -0.69921875, 1.0439453125, 2.447265625, -1.57421875, -0.3125, -1.8125, 0.280029296875, 0.39794921875, -0.0645751953125, -0.30859375, -0.7841796875, -0.1285400390625, 0.461669921875, 0.19482421875, 1.142578125, 0.64697265625, 0.09906005859375, 0.70166015625, 1.455078125, 0.49462890625, -0.2359619140625, 1.6259765625, -0.368896484375, 0.89111328125, 0.8046875, 1.0458984375, 0.2822265625, 1.837890625, 1.2724609375, -0.13037109375, 0.1497802734375, -0.10833740234375, -0.56787109375, -0.56201171875, -0.64794921875, 0.08343505859375, 1.365234375, -0.73095703125, -0.058197021484375, -0.2059326171875, 0.462646484375, -0.10064697265625, -0.28076171875, 1.7490234375, -0.031829833984375, -0.30224609375, 1.4912109375, -1.1650390625, -2.466796875, 0.23779296875, 0.76025390625, 1.8935546875, -1.7138671875, 0.005527496337890625, 0.039825439453125, -0.487060546875, 0.9130859375, -0.483154296875, 0.174560546875, 1.3671875, 2.05859375, -0.95556640625, 0.7138671875, -0.5830078125, -0.67822265625, -0.236572265625, -0.787109375, 0.049530029296875, -0.392578125, -0.365478515625, -0.09619140625, 0.13134765625, -1.986328125, 1.8759765625, 0.134765625, 0.422119140625, 1.638671875, 1.40234375, -1.2275390625, 1.796875, 1.4521484375, 0.088623046875, 0.57568359375, -2.064453125, 0.49267578125, 1.787109375, 0.445556640625, -1.576171875, -1.1962890625, -0.6884765625, -0.56787109375, -0.1700439453125, 0.366455078125, -0.55224609375, -1.6708984375, -2.2265625, -1.2509765625, 0.70947265625, -0.75390625, 0.58837890625, 0.986328125, 0.76904296875, 0.8271484375, -0.53076171875, -0.96630859375, 0.48876953125, 0.03692626953125, 0.0205841064453125, 0.073974609375, 1.6591796875, 0.072021484375, 1.09765625, -0.81787109375, 0.6884765625, 0.14794921875, 0.517578125, -1.8583984375, -0.48095703125, 0.11865234375, 0.3212890625, 0.3681640625, 0.208984375, -0.89013671875, 2.046875, 0.46337890625, -0.4384765625, 1.201171875, -0.2484130859375, -1.3701171875, -1.3662109375, -0.1063232421875, 0.67138671875, -0.6083984375, -0.84521484375, -0.50927734375, -0.80810546875, 0.06884765625, 0.5966796875, 0.90234375, -1.158203125, -1.6494140625, -1.16796875, -0.8349609375, -0.5634765625, 0.7109375, -0.5009765625, -0.833984375, 0.26123046875, -0.7822265625, -0.2685546875, -0.32177734375, -1.82421875, 1.8896484375, 0.04339599609375, -0.6376953125, -1.283203125, 0.8525390625, 0.464111328125, 0.157958984375, -1.375, 0.60693359375, -1.4267578125, 1.125, 1.2841796875, -0.10101318359375, 0.4208984375, -1.0986328125, 0.72998046875, -0.80712890625, 0.1923828125, -1.443359375, 1.740234375, -0.30810546875, 0.7685546875, -0.44873046875, -2.4921875, -0.55517578125, -0.46484375, 0.1082763671875, 0.4755859375, -0.755859375, -1.283203125, -0.5625, 0.02984619140625, -0.97119140625, -0.3876953125, -0.806640625, 0.0010061264038085938, 0.3017578125, 0.85205078125, -0.37744140625, -1.431640625, -1.01171875, 0.1298828125, 0.81103515625, 1.4775390625, -0.2861328125, 0.5791015625, 0.41796875, -0.58447265625, 3.087890625, 0.30810546875, 0.49169921875, -0.47509765625, 0.82275390625, -0.7685546875, -0.96826171875, 0.96337890625, 0.61181640625, 1.0478515625, -0.027374267578125, 0.261474609375, -2.162109375, 0.58740234375, 0.86328125, 0.69189453125, -1.3525390625, -1.4072265625, -1.150390625, 0.08038330078125, 0.86962890625, -1.8896484375, -0.236328125, -3.173828125, 0.299072265625, -0.72705078125, -1.259765625, 0.376708984375, 0.3154296875, 0.21484375, -1.34375, -0.66357421875, 0.463134765625, -1.2314453125, 0.4482421875, 0.8759765625, -1.099609375, -0.60498046875, 0.468505859375, -1.037109375, 0.1873779296875, -1.0703125, 0.3349609375, 0.77392578125, -0.99365234375, 0.24267578125, -0.26806640625, 1.7724609375, 2.080078125, 2.49609375, 0.90869140625, -0.759765625, -0.11956787109375, -0.82666015625, -0.63232421875, 0.392333984375, -0.09173583984375, 1.4658203125, 0.521484375, -1.4306640625, -0.53564453125, -0.286376953125, 0.347900390625, -0.353759765625, 0.40625, -1.6044921875, 0.8486328125, 1.544921875, 0.356689453125, 0.76513671875, 1.447265625, 0.60205078125, 0.75341796875, 0.87890625, 0.52734375, 0.251220703125, -0.202880859375, -0.6572265625, 0.243896484375, 1.1279296875, 0.81103515625, -0.01396942138671875, -0.348876953125, 0.352294921875, -0.21728515625, -0.006183624267578125, 1.1201171875, 1.6552734375, 0.00818634033203125, -0.3427734375, -1.81640625, -0.0948486328125, -0.06451416015625, 1.267578125, 0.017425537109375, 1.046875, 0.2587890625, -0.64892578125, -1.2802734375, 0.833984375, 1.244140625, 1.1240234375, 0.123046875, 0.7705078125, 0.77978515625, -0.33251953125, 0.68798828125, 0.5166015625, 1.62890625, 0.27978515625, 0.66943359375, -0.427734375, -0.020782470703125, 0.325439453125, 0.1868896484375, 0.41259765625, -0.0909423828125, -0.369140625, -0.416748046875, 0.3857421875, 0.485107421875, 0.72705078125, 0.60302734375, 1.443359375, 0.41064453125, -0.57763671875, -1.30859375, -1.2978515625, 0.51953125, -0.0177764892578125, 0.491943359375, 0.7763671875, -0.3115234375, 0.80126953125, -2.005859375, 0.82080078125, 0.47607421875, 0.53662109375, -1.8310546875, -2.134765625, 0.444091796875, -0.1265869140625, -1.2109375, 0.986328125, -0.9423828125, 0.67822265625, 0.352294921875, 0.8251953125, 1.1826171875, 1.294921875, -0.390625, 1.2255859375, -1.6181640625, 0.257080078125, -0.08770751953125, -1.4306640625, -1.7685546875, 0.65087890625, -1.6962890625, 0.2442626953125, 0.1510009765625, 1.173828125, -0.57177734375, -0.6796875, 1.005859375, 0.6953125, -0.578125, 0.439208984375, -0.388671875, 0.2890625, -0.314453125, -0.033935546875, 0.27978515625, -0.8466796875, -0.11822509765625, 0.5361328125, 0.6357421875, 2.326171875, -0.681640625, 1.171875, -0.008758544921875, -0.1748046875, -1.1357421875, 0.779296875, 0.2486572265625, -0.755859375, 1.1513671875, -0.845703125, 2.111328125, -0.0814208984375, 1.423828125, -0.69580078125, 0.880859375, 0.344970703125, -1.095703125, 1.5947265625, -0.1890869140625, -1.2080078125, 0.6259765625, -0.64404296875, -0.13916015625, 0.16845703125, 1.0087890625, 0.61962890625, 0.7646484375, 1.0029296875, -0.19189453125, -0.65576171875, -0.583984375, -0.2362060546875, 1.2802734375, -0.1514892578125, -0.40478515625, -1.0302734375, 1.6611328125, 0.275390625, 0.447998046875, -1.3583984375, -0.9306640625, 0.5517578125, 0.87841796875, -1.46484375, 0.703125, -0.86572265625, -0.4619140625, 1.5546875, -0.88623046875, 1.1455078125, -2.474609375, 0.5654296875, 1.1748046875, -0.509765625, -0.18115234375, 0.4716796875, -1.724609375, 1.419921875, 0.7060546875, 0.36962890625, 1.30078125, 1.5009765625, -0.332275390625, 0.007671356201171875, 1.08984375, 0.603515625, -1.244140625, -1.373046875, -1.25, 1.4580078125, -0.73583984375, 0.2099609375, -0.94775390625, 1.63671875, -0.86279296875, 0.97900390625, 0.5498046875, -1.2900390625, -1.615234375, -0.654296875, -0.2281494140625, -1.478515625, 0.72607421875, 0.392578125, 1.126953125, -0.9375, -1.0498046875, 0.01306915283203125, -0.444091796875, 0.14306640625, 0.01611328125, -1.4921875, 0.1478271484375, -0.1068115234375, 0.1795654296875, 1.2353515625, 0.99560546875, 1.166015625, -0.5771484375, 0.309814453125, 0.67236328125, -2.513671875, 0.5380859375, 2.921875, -0.400634765625, 0.1649169921875, 0.947265625, -0.93603515625, 1.07421875, -0.47705078125, -0.74462890625, -0.609375, 0.46435546875, 0.8291015625, 1.1650390625, 0.5771484375, 0.2161865234375, 0.61669921875, -1.955078125, 0.05841064453125, -0.6923828125, 0.57275390625, 1.556640625, -1.337890625, 1.4296875, -0.28515625, -0.3076171875, 0.158203125, -0.6953125, 0.85400390625, -0.70068359375, -0.318359375, -0.97412109375, 1.2783203125, 1.2001953125, -0.357666015625, -1.16015625, 0.349609375, 1.7294921875, -0.5126953125, 1.126953125, 1.736328125, 1.05859375, -1.697265625, 2.05078125, 1.2451171875, 0.68603515625, 1.2392578125, 1.8310546875, -1.2236328125, 0.280029296875, 0.2335205078125, 0.70751953125, 0.039642333984375, 2.82421875, 1.017578125, 0.72216796875, -0.367919921875, -0.03802490234375, -0.3017578125, 1.5703125, 1.0224609375, -1.294921875, 0.0439453125, 0.54541015625, 0.0202178955078125, 1.8583984375, -0.56884765625, -0.099853515625, -0.58837890625, 1.078125, -0.59228515625, 0.8076171875, -0.09124755859375, -0.9130859375, -0.2476806640625, 1.048828125, 2.505859375, -0.931640625, 0.69970703125, 0.218017578125, -0.416259765625, -0.0178070068359375, 1.2216796875, -1.140625, 1.6455078125, -1.0947265625, -0.3955078125, -0.5263671875, -1.4931640625, -0.54052734375, 0.1370849609375, -0.658203125, 1.4365234375, -1.2919921875, 0.5654296875, -0.12249755859375, -1.2578125, -1.8818359375, 0.16796875, -0.382080078125, 1.5625, -1.33984375, 0.01517486572265625, -0.63330078125, 0.96728515625, 0.875, -0.312255859375, -0.1190185546875, 0.6396484375, 0.24609375, 1.9423828125, 0.666015625, -0.2374267578125, -1.09375, -2.439453125, 0.66845703125, 0.187255859375, 1.87890625, 0.249267578125, 1.7548828125, 0.30078125, -0.4619140625, 0.947265625, -1.12109375, 0.8828125, -0.08477783203125, 0.732421875, -0.86083984375, -1.884765625, -1.5458984375, 1.2763671875, -0.426025390625, -0.046234130859375, 0.1595458984375, 1.6611328125, 0.1029052734375, 0.0714111328125, -0.452880859375, 0.32421875, -0.8076171875, 0.96875, -1.111328125, 0.51171875, -0.9921875, -1.6162109375, -0.76171875, -0.6240234375, 0.576171875, 0.95458984375, -0.477783203125, -0.66064453125, 0.85498046875, -0.95703125, -0.430419921875, 0.765625, -0.2578125, 0.87451171875, 2.341796875, 0.311767578125, 0.611328125, -1.24609375, 0.12042236328125, 1.5185546875, -0.00937652587890625, 0.80908203125, -0.77001953125, -0.01218414306640625, 0.42724609375, -0.0810546875, 0.8515625, -0.91748046875, 1.576171875, -0.142578125, -0.98388671875, 0.74267578125, 0.6279296875, 1.0498046875, -0.349609375, 0.06689453125, 0.74365234375, -0.1475830078125, -2.0625, 0.300048828125, -0.8759765625, -3.1796875, -0.23291015625, 0.1092529296875, -0.3896484375, -0.39013671875, -0.3994140625, 0.9013671875, 0.89453125, 0.560546875, 0.1531982421875, -0.44482421875, -0.457763671875, 0.85498046875, 0.45556640625, -1.5380859375, -0.053619384765625, -2.078125, -0.994140625, -0.92333984375, 0.474609375, -0.6640625, -0.7412109375, -2.529296875, -0.95458984375, -1.5078125, -0.3818359375, 0.91357421875, 1.8525390625, 1.806640625, -0.52587890625, 0.17529296875, -1.470703125, -1.1669921875, 0.45703125, -1.9501953125, 0.7646484375, 1.0029296875, 1.703125, -0.4658203125, -0.54931640625, 1.71875, -0.3046875, -1.2080078125, -1.1318359375, -0.421875, -1.203125, -1.4873046875, -0.2027587890625, 0.51123046875, 1.384765625, 1.193359375, -1.29296875, 0.61376953125, -0.31689453125, 0.65478515625, 0.1468505859375, 0.85888671875, 1.8349609375, 0.2054443359375, -0.51953125, -0.26318359375, -1.8349609375, -0.7080078125, -1.1474609375, -0.39501953125, 0.0251617431640625, -1.8427734375, -0.049407958984375, -0.49365234375, 0.5625, -0.07733154296875, 0.66796875, 0.5302734375, -0.50537109375, 2.154296875, 0.78466796875, 1.375, 0.34033203125, 1.15234375, 0.1448974609375, 0.82373046875, 0.252197265625, 0.75341796875, 1.6201171875, 0.1993408203125, 0.35009765625, -1.2763671875, 0.65966796875, -0.2744140625, 0.5224609375, -0.75390625, 0.2252197265625, -0.004070281982421875, -0.6884765625, -0.69482421875, 0.1092529296875, 0.591796875, 0.78466796875, 0.2489013671875, 0.30908203125, 1.54296875, 0.303955078125, -0.80419921875, 0.71240234375, -2.736328125, 1.076171875, 0.318603515625, -1.4443359375, -0.6474609375, -0.31494140625, 1.37890625, 0.982421875, 0.483642578125, 0.46435546875, 0.59521484375, 0.7294921875, 0.83984375, -0.69921875, -0.01122283935546875, 0.0906982421875, -0.509765625, -1.26171875, -1.462890625, 0.232177734375, -1.1884765625, -0.438720703125, -0.453369140625, 0.287353515625, 1.345703125, -0.1275634765625, 0.39794921875, 1.486328125, 0.1998291015625, 0.336181640625, 1.193359375, -1.2265625, 1.5263671875, -0.037750244140625, -0.3876953125, -1.50390625, 0.80712890625, 1.625, -1.3115234375, 0.68408203125, -0.62353515625, 0.0248260498046875, -1.427734375, -1.0498046875, 1.189453125, 0.81884765625, 0.54248046875, 0.5751953125, -1.4541015625, 0.062744140625, 1.3583984375, -0.76025390625, 0.87841796875, 0.93310546875, 0.25390625, -1.419921875, -0.15673828125, 1.2158203125, -0.5986328125, -1.4833984375, -1.7216796875, 0.5224609375, 0.79833984375, 0.6181640625, 0.2255859375, 0.2156982421875, -0.485107421875, 0.34228515625, -0.71142578125, 1.646484375, 0.4833984375, -0.60595703125, -1.8505859375, -0.66162109375, 0.94189453125, 0.5830078125, -0.130859375, 0.397705078125, 3.087890625, -0.763671875, 1.6298828125, 0.693359375, 0.36962890625, -0.2149658203125, -0.24951171875, -1.1318359375, 0.89404296875, -1.052734375, 0.92431640625, -0.0933837890625, -0.63232421875, -1.3935546875, 0.77587890625, 0.398193359375, 0.02374267578125, 0.908203125, 0.77978515625, -1.556640625, 0.7802734375, -0.230712890625, -0.85986328125, 0.2357177734375, -0.59716796875, 0.88720703125, 0.70751953125, -0.65771484375, -0.884765625, 1.359375, 0.10516357421875, -1.3388671875, 0.759765625, -1.2705078125, -0.771484375, 0.340087890625, 0.2130126953125, -0.9423828125, -0.218505859375, -0.430908203125, -0.1973876953125, 1.7294921875, 0.07489013671875, -0.450439453125, -1.7060546875, 0.54931640625, 0.041778564453125, 1.345703125, -0.92138671875, 0.19384765625, 0.7880859375, 1.09375, -0.4560546875, 1.1640625, 0.69677734375, 0.352294921875, -0.857421875, 2.181640625, 0.281982421875, 0.162353515625, -0.290283203125, 0.4462890625, 0.0855712890625, 0.10260009765625, 0.2464599609375, 1.517578125, 0.30517578125, -2.5625, 1.037109375, 0.134765625, 0.7705078125, 0.1844482421875, 0.07958984375, 0.47216796875, 0.1361083984375, -3.333984375, 0.0017480850219726562, -0.11248779296875, -0.6845703125, -0.365234375, -1.134765625, 1.3583984375, 0.0546875, 0.6845703125, -1.224609375, -1.75, -0.250732421875, -0.794921875, 0.43994140625, -1.7197265625, 0.85595703125, -1.865234375, -1.2841796875, -0.34521484375, 0.28857421875, 0.6064453125, 0.331298828125, -1.8857421875, -1.109375, 0.410400390625, -0.79443359375, 0.6015625, -0.5859375, 2.296875, -0.5478515625, -0.47412109375, 0.373291015625, 0.1473388671875, -0.66015625, 1.8544921875, 0.07330322265625, -0.1932373046875, -1.78125, -0.60498046875, -2.111328125, -0.256103515625, 2.263671875, -1.5546875, 1.228515625, 0.91064453125, -0.0200042724609375, 1.6806640625, 0.93115234375, -1.1962890625, -0.330078125, 0.37255859375, 1.396484375, 0.2154541015625, 1.490234375, 0.05084228515625, 0.54541015625, 0.45654296875, 2.458984375, -1.6455078125, 0.7333984375, 1.77734375, 0.298583984375, 0.396728515625, 2.728515625, -0.39208984375, -0.98095703125, 0.4619140625, 0.62939453125, 0.58935546875, 0.78466796875, -0.56591796875, 0.5048828125, 2.00390625, 0.39208984375, 1.509765625, -0.5400390625, -2.05078125, 1.1728515625, -1.01171875, 0.155517578125, -0.0582275390625, 0.83203125, 0.09564208984375, -0.16259765625, -0.415771484375, -1.1669921875, -0.66259765625, -2.587890625, 0.11541748046875, -0.6826171875, -0.7646484375, -2.912109375, 0.448974609375, 0.80712890625, 0.50341796875, 0.14111328125, -1.9833984375, -0.2132568359375, 1.876953125, -1.0634765625, 0.476318359375, -0.490966796875, 0.05645751953125, -1.7568359375, 2.43359375, 0.7138671875, -1.1767578125, -0.381591796875, -1.9560546875, -0.70556640625, 0.658203125, 1.0615234375, 0.85693359375, 2.57421875, 1.455078125, -0.8671875, -0.0501708984375, -0.433349609375, -0.05078125, 1.673828125, -0.11004638671875, -0.059661865234375, 1.2451171875, 1.8271484375, 1.322265625, -0.96826171875, -0.228759765625, -0.2149658203125, 0.791015625, 0.76611328125, -1.103515625, -0.63037109375, -0.29296875, -0.3779296875, 0.845703125, -0.5546875, -0.4990234375, -0.205322265625, 2.31640625, -0.55615234375, -1.2275390625, 0.71435546875, 0.53076171875, -0.26904296875, 1.0947265625, 0.03533935546875, 1.2587890625, 0.533203125, 0.40283203125, -1.203125, -0.99365234375, 0.853515625, -0.86279296875, 0.75048828125, 0.48828125, -0.02471923828125, -1.25, 0.38232421875, -1.6005859375, 0.09320068359375, 0.221435546875, -1.3662109375, -0.79443359375, -0.09356689453125, 0.52294921875, 1.11328125, 1.548828125, -0.52978515625, -0.7734375, -1.3642578125, -0.1724853515625, 0.7578125, 1.00390625, 1.2861328125, -0.63330078125, 1.3955078125, -0.08984375, -0.21826171875, 0.322998046875, 0.75341796875, -0.90625, 0.82568359375, -0.479248046875, 0.69140625, 0.61181640625, -1.14453125, 0.35546875, 0.89404296875, -0.759765625, -0.83544921875, 0.1279296875, 0.2132568359375, 0.47119140625, -1.0009765625, 0.9189453125, 0.43359375, 0.68798828125, -1.9775390625, -0.083984375, 0.59375, -0.619140625, 0.437744140625, -0.436767578125, 0.348876953125, 1.015625, -1.4765625, 0.82421875, 1.359375, -1.1201171875, 0.007595062255859375, 0.625, -1.0205078125, 0.491943359375, -0.70947265625, -0.27490234375, -0.63427734375, 1.638671875, 0.802734375, 0.1746826171875, 0.65185546875, -1.5439453125, -0.73095703125, -1.1796875, 2.55078125, -0.9423828125, -0.1241455078125, 1.677734375, 0.58837890625, 0.74462890625, -0.47705078125, -0.430419921875, 1.1025390625, 0.443359375, -0.08441162109375, -0.59619140625, -0.394287109375, -0.6826171875, 0.371337890625, 0.152587890625, 0.33837890625, -1.349609375, 0.638671875, 0.45654296875, -0.51611328125, 0.43408203125, -1.296875, 0.1474609375, 0.7041015625, 2.267578125, -1.3505859375, -1.748046875, 2.158203125, 0.1175537109375, -0.8125, -0.483154296875, 1.2802734375, 0.212646484375, -0.404052734375, -0.923828125, -0.6923828125, -0.96240234375, -1.787109375, -1.0888671875, -1.52734375, 0.16650390625, -0.99169921875, 1.1572265625, 0.441650390625, 0.147216796875, -1.4755859375, -0.29736328125, -0.01050567626953125, 0.06793212890625, 0.2156982421875, -1.177734375, 0.79541015625, -0.92333984375, 0.57568359375, -0.1953125, -0.393798828125, -1.6826171875, 0.2919921875, -0.425048828125, 0.1890869140625, -0.09722900390625, -0.01220703125, -0.75634765625, -1.1416015625, 1.6787109375, 0.6064453125, -0.92529296875, 1.265625, 0.826171875, -1.115234375, 1.23828125, -0.986328125, 0.87060546875, 2.478515625, 0.12646484375, 0.08843994140625, -0.351318359375, 0.49755859375, 0.007640838623046875, 0.38525390625, 0.654296875, -0.30517578125, 0.0726318359375, -0.73681640625, 0.78759765625, 1.0048828125, -0.354248046875, -1.7978515625, -0.7119140625, 0.853515625, -1.5810546875, -0.06939697265625, 1.7333984375, 0.06414794921875, -1.1328125, -0.9921875, 0.52392578125, -0.498046875, 1.169921875, 1.2822265625, -0.41015625, -0.61474609375, -1.23046875, 0.8505859375, -0.15185546875, -0.237060546875, -0.53271484375, 0.85205078125, -0.93701171875, -0.0228118896484375, -0.45849609375, 2.345703125, -0.85498046875, 0.2413330078125, -2.123046875, 1.294921875, -0.59033203125, 1.1015625, 1.6025390625, 1.9580078125, -1.8623046875, -0.78076171875, 0.67333984375, -0.9853515625, 0.11016845703125, -0.7060546875, -0.403076171875, 0.383544921875, -0.033477783203125, 1.291015625, -1.0107421875, -0.59912109375, 0.2491455078125, 2.72265625, 0.138427734375, -1.51953125, 0.6806640625, -0.167236328125, 1.3173828125, -0.75830078125, -1.23828125, -0.517578125, 2.505859375, -1.1396484375, 0.71240234375, -0.30322265625, -1.0126953125, 1.0087890625, -1.078125, -1.380859375, 1.638671875, 0.607421875, 0.455810546875, -1.18359375, -0.04705810546875, -0.2418212890625, 0.6396484375, 0.25927734375, 0.155517578125, -0.09466552734375, -1.3232421875, 0.306640625, 1.001953125, -0.393310546875, -1.029296875, -1.271484375, 0.505859375, -0.1761474609375, 0.397705078125, 0.62890625, -1.32421875, 0.51611328125, 0.51416015625, 0.89013671875, 0.304931640625, 0.35498046875, -0.8076171875, 0.69580078125, 1.4794921875, 0.12017822265625, 0.76611328125, 0.2462158203125, -0.52880859375, -0.04534912109375, -0.76611328125, -0.295166015625, 1.03125, -1.001953125, 2.537109375, 0.818359375, -0.53759765625, -1.453125, -1.0302734375, 0.578125, -0.52734375, -0.233154296875, 1.0595703125, -0.4462890625, 0.09521484375, 0.23974609375, -0.58544921875, -1.626953125, -0.98291015625, -0.52685546875, 0.78515625, 0.98681640625, 0.1630859375, 0.2215576171875, 0.235107421875, 0.0187835693359375, 0.31982421875, -0.016387939453125, -1.48828125, 1.595703125, -0.97119140625, 1.50390625, 0.67041015625, -0.77978515625, -0.50146484375, 0.1396484375, 0.398681640625, -0.89111328125, -0.9775390625, -0.08062744140625, -0.064697265625, 1.2294921875, 0.65087890625, -0.423095703125, -0.389892578125, -0.256591796875, 0.02587890625, -0.60888671875, 0.164794921875, -0.78662109375, -1.3994140625, -1.0107421875, -0.89892578125, 0.04913330078125, 1.1650390625, 0.91064453125, 1.1708984375, 1.0888671875, -0.0693359375, -0.7451171875, 0.0262603759765625, 1.4169921875, -0.5576171875, 0.263916015625, 0.049346923828125, 1.8427734375, -0.217529296875, 0.3603515625, 1.2724609375, -0.84765625, 1.0888671875, -0.9736328125, 0.50439453125, 0.6279296875, -0.98388671875, -0.466796875, 2.1015625, -0.046356201171875, 0.1490478515625, 1.0654296875, -0.281494140625, -0.2548828125, -0.1461181640625, 1.1162109375, 0.5380859375, -0.5615234375, -2.044921875, 0.262451171875, -0.67041015625, -0.2064208984375, 0.87548828125, -0.0187530517578125, 0.27783203125, -0.1165771484375, -1.408203125, 0.357666015625, -1.0048828125, 0.9609375, 0.57568359375, -0.329345703125, 0.29736328125, -0.47607421875, 0.87353515625, -2.283203125, -0.2353515625, -1.017578125, -0.0755615234375, 0.1834716796875, 0.951171875, 0.88818359375, -0.1485595703125, -0.89208984375, 0.002162933349609375, 0.9130859375, -0.03326416015625, -0.91015625, 0.37548828125, 1.044921875, 1.66796875, 0.0899658203125, -1.9072265625, -0.32080078125, -0.177001953125, 0.06219482421875, 0.308837890625, -1.1533203125, 0.34326171875, -0.5849609375, -1.37109375, -2.25390625, -0.45458984375, 0.2763671875, 1.115234375, -1.6796875, 0.1121826171875, 1.1201171875, -2.423828125, 0.8369140625, 0.319580078125, 0.2064208984375, 0.342529296875, -0.6787109375, -0.37548828125, -0.955078125, 0.7548828125, 0.439453125, 0.356689453125, -0.0419921875, -0.206787109375, 0.6689453125, -0.1529541015625, 0.09124755859375, -0.286376953125, -0.873046875, 0.74462890625, -0.794921875, 0.1651611328125, -0.0654296875, 0.1259765625, 0.08856201171875, -0.595703125, 1.7353515625, -0.603515625, 1.1884765625, 0.15380859375, 0.98779296875, -1.169921875, -1.248046875, 0.9765625, 2.271484375, 1.06640625, -1.4189453125, -0.67236328125, 1.36328125, 0.382568359375, -1.58984375, 0.98828125, 1.416015625, -0.60791015625, 0.7099609375, 0.1539306640625, 1.84765625, -1.0087890625, -0.095458984375, 0.06365966796875, 0.0306396484375, 1.1064453125, 0.08404541015625, 2.37890625, -0.68115234375, -0.861328125, -2.015625, 1.0185546875, 1.087890625, 0.4501953125, 0.689453125, -0.46435546875, 1.830078125, 2.267578125, 0.431396484375, 1.0419921875, 0.191650390625, 1.3720703125, -0.2841796875, -2.21484375, 1.1416015625, -1.2353515625, -0.1431884765625, 0.01059722900390625, 0.057281494140625, -0.08941650390625, 0.8583984375, 0.314453125, 0.744140625, 0.07147216796875, 0.45166015625, -0.7578125, -1.4912109375, 0.6572265625, 0.9462890625, 0.2156982421875, -1.9677734375, -0.580078125, 0.040069580078125, 0.5146484375, 1.033203125, -1.2109375, 1.904296875, 0.77294921875, 0.43310546875, 1.4287109375, -0.51416015625, 1.3134765625, -0.171630859375, -0.66650390625, 1.4228515625, -0.83056640625, -1.0693359375, -1.9560546875, 0.55029296875, 0.2587890625, -0.7333984375, -0.4599609375, -1.1806640625, -1.29296875, -0.216064453125, -0.509765625, -0.308349609375, 0.5205078125, -0.54931640625, -0.35498046875, -1.6123046875, 1.2548828125, 0.099853515625, 0.202392578125, -0.71728515625, -2.353515625, -0.6513671875, 2.21484375, 0.4384765625, 0.08111572265625, 1.158203125, 0.564453125, 0.0164794921875, -0.92724609375, 1.19140625, -0.288818359375, 2.244140625, 1.0234375, 0.335693359375, 0.5302734375, 1.9775390625, -0.40771484375, -0.48388671875, -1.8525390625, 1.0126953125, -0.546875, 0.5029296875, 1.091796875, 2.892578125, -0.480712890625, 0.71826171875, -0.252197265625, -1.533203125, -1.8408203125, -0.09320068359375, 0.180908203125, -0.11572265625, 0.75390625, -2.287109375, 1.0732421875, 0.892578125, -2.087890625, -0.35498046875, -1.673828125, 0.448974609375, -1.21875, 0.0546875, -0.73681640625, 0.0626220703125, 0.5439453125, -0.72998046875, -0.08258056640625, -0.380615234375, -0.244384765625, 0.97900390625, -0.258056640625, -0.271240234375, 1.673828125, -0.25146484375, 0.509765625, -0.0153045654296875, -0.22412109375, -0.51123046875, 1.4169921875, -0.33203125, 0.185791015625, 0.8408203125, 0.64453125, 0.380126953125, 0.04730224609375, -0.337158203125, 0.33544921875, -0.04229736328125, 0.1956787109375, -0.38720703125, 1.9365234375, -0.09210205078125, -1.8955078125, -0.2607421875, -0.346923828125, -1.1767578125, 0.21240234375, 1.087890625, 1.8974609375, 0.78369140625, -0.68408203125, 1.125, -0.9404296875, -0.478515625, -0.90771484375, 1.9853515625, -0.1829833984375, -0.6552734375, -0.94580078125, -0.355712890625, 0.55859375, -0.206298828125, 0.10699462890625, -0.55419921875, -0.8876953125, -0.1214599609375, 0.279052734375, -0.8232421875, -0.409912109375, 2.9453125, -1.8818359375, 0.054840087890625, 0.451904296875, -1.14453125, 1.2216796875, -0.46533203125, -0.105224609375, -1.4912109375, 0.0816650390625, 0.19580078125, -1.748046875, 1.0166015625, 0.5068359375, -0.30615234375, 1.40625, -0.40576171875, -1.14453125, 0.133544921875, -0.66943359375, -1.09765625, -0.3818359375, 0.93701171875, 1.18359375, 1.1708984375, 0.090576171875, -0.451171875, -1.0302734375, 1.3525390625, 1.357421875, -1.166015625, -0.7734375, 0.9228515625, -0.2420654296875, 1.5732421875, 0.6611328125, 0.334716796875, -0.2069091796875, 1.5849609375, -1.451171875, -0.46630859375, -0.70068359375, -0.93701171875, 1.1748046875, -1.1416015625, 0.464599609375, -1.490234375, -0.97607421875, 0.1370849609375, -0.6171875, -0.27685546875, -1.751953125, -0.0084075927734375, -0.28955078125, 0.03509521484375, 1.4912109375, -0.74951171875, 0.015655517578125, -0.323974609375, -0.8203125, 2.044921875, -0.306640625, 1.2763671875, -0.62548828125, -0.033905029296875, -1.3935546875, 0.94775390625, -1.8720703125, 1.0693359375, 0.4052734375, -1.365234375, -1.7646484375, 0.343994140625, -1.744140625, 1.1259765625, -0.39990234375, 0.56005859375, -0.36962890625, -0.189697265625, -0.8505859375, 1.0068359375, -1.841796875, -0.292724609375, 1.0322265625, -1.7275390625, 2.103515625, 0.1627197265625, 0.218017578125, 1.0126953125, 0.96826171875, -0.153564453125, -1.03125, -0.70556640625, 1.013671875, -2.267578125, 0.287841796875, 1.37109375, -1.283203125, -1.3291015625, -1.6787109375, 0.8974609375, -1.5703125, -0.0926513671875, 0.8525390625, 1.06640625, 1.76953125, 1.478515625, -1.07421875, -1.6376953125, 1.5986328125, 0.673828125, -0.197509765625, -0.4140625, 1.8212890625, 0.1695556640625, -1.6669921875, -0.201171875, 0.355224609375, 0.60791015625, -0.3310546875, -0.00949859619140625, -1.73828125, -0.1131591796875, -1.119140625, -1.015625, 0.16796875, 0.454345703125, 1.291015625, 0.061737060546875, 0.7373046875, -1.65625, 0.638671875, -0.6220703125, 1.2060546875, -0.26318359375, 1.2236328125, -0.1800537109375, -0.2225341796875, -0.284912109375, 0.0692138671875, 0.467529296875, -1.7236328125, 0.451416015625, -0.982421875, -0.798828125, -0.8798828125, -0.9345703125, -1.7373046875, 1.1044921875, -0.95458984375, 0.91357421875, -0.77197265625, -0.4775390625, 0.826171875, -0.163818359375, -0.4189453125, -0.65771484375, -0.1043701171875, 1.3701171875, 0.472900390625, 0.57568359375, -1.1015625, 1.087890625, 1.205078125, 0.27734375, -0.3193359375, -0.102294921875, 0.4287109375, 0.201904296875, -0.7138671875, -1.033203125, -0.96875, -0.99365234375, -0.325927734375, 0.1536865234375, 1.287109375, 0.408447265625, 0.1817626953125, 0.8408203125, -1.0966796875, 1.2900390625, 1.4248046875, 0.69677734375, -0.75048828125, 0.89599609375, -0.1746826171875, 1.107421875, -1.4453125, -0.6064453125, -0.0078582763671875, 0.1015625, -2.162109375, -0.6337890625, 0.34521484375, -0.40576171875, 0.150390625, 0.486572265625, 0.377685546875, 0.73193359375, 0.468505859375, -1.0888671875, -0.251953125, -0.06304931640625, -1.4296875, -0.51611328125, -0.1217041015625, 0.54296875, 0.11981201171875, -0.64697265625, 0.236572265625, -0.2020263671875, 1.423828125, -0.85205078125, 1.859375, 0.43505859375, 0.55517578125, 1.7177734375, 0.19091796875, 0.1845703125, 1.1259765625, 0.05303955078125, 0.156982421875, -0.73095703125, -0.302978515625, 0.0296173095703125, 1.40625, -0.970703125, -0.80517578125, 0.0203704833984375, 1.5166015625, 1.181640625, 1.873046875, 1.7919921875, -0.58642578125, -0.94921875, -0.986328125, -0.90380859375, -0.2406005859375, 1.5537109375, -1.5380859375, 0.99169921875, 0.2568359375, 1.34765625, 1.9228515625, 0.1593017578125, -2.037109375, 0.76171875, 1.1796875, 1.3837890625, -1.31640625, 0.2054443359375, -0.32421875, -0.2462158203125, 1.28515625, -1.166015625, 0.02508544921875, -0.055328369140625, -0.497314453125, 0.9111328125, 1.2412109375, 1.0859375, 0.8603515625, 0.85107421875, -1.28125, -1.0390625, -0.6015625, 0.87841796875, -0.6826171875, 2.96875, 2.537109375, 2.35546875, -1.115234375, -0.1527099609375, -0.0550537109375, -0.5654296875, 1.724609375, 0.50341796875, 0.548828125, 0.86962890625, -0.2880859375, -0.8154296875, 0.189697265625, -1.3037109375, -0.23291015625, -0.94921875, 1.0810546875, -0.1990966796875, 0.5126953125, -0.97119140625, -0.802734375, -0.52490234375, -1.111328125, 0.740234375, -1.130859375, 0.08233642578125, 1.537109375, 1.130859375, 1.5146484375, 0.6025390625, 0.1077880859375, -0.121337890625, 1.171875, 0.66650390625, -0.83837890625, 0.77001953125, 0.3671875, -0.416015625, 0.34033203125, -0.56396484375, 1.2177734375, 0.355712890625, 0.4892578125, 0.53369140625, 1.13671875, 1.982421875, -0.11224365234375, 0.64013671875, -0.56982421875, 0.8740234375, 0.6044921875, -0.26708984375, -0.4951171875, 0.134765625, -0.0633544921875, 2.689453125, 1.5830078125, -0.11175537109375, 0.1898193359375, 1.2529296875, -0.3173828125, 1.142578125, 1.39453125, -1.3251953125, -0.83154296875, -2.02734375, -1.20703125, -0.4453125, 0.458740234375, -0.0965576171875, 0.492431640625, 1.9267578125, -0.36474609375, 0.042144775390625, 1.30859375, -0.06439208984375, 0.4189453125, -1.2998046875, -0.406005859375, 0.75537109375, -0.1671142578125, 1.619140625, 2.509765625, 0.08294677734375, 0.02520751953125, -0.02850341796875, -1.0634765625, -0.45849609375, 2.490234375, 0.055419921875, 0.01439666748046875, -0.48046875, -1.90234375, 1.009765625, 0.51171875, -0.7490234375, 0.29052734375, -0.58251953125, 0.072509765625, 0.68310546875, -0.2347412109375, -0.54931640625, 0.036468505859375, 0.2445068359375, 1.4609375, -0.326904296875, -0.253173828125, -0.0206756591796875, 2.046875, -1.7255859375, 0.87841796875, 0.76953125, 0.8642578125, -0.97802734375, 0.382080078125, 0.397216796875, -0.019805908203125, 0.279052734375, -0.07501220703125, -0.290771484375, -0.5419921875, 0.40625, -0.1788330078125, 0.8994140625, -0.904296875, -0.032623291015625, 0.65185546875, -1.330078125, -2.3046875, -0.0694580078125, 0.5908203125, 0.84326171875, -0.02020263671875, -1.3232421875, 0.67578125, -1.6279296875, 0.96630859375, -0.282470703125, 1.3271484375, 1.9443359375, 0.875, -0.1864013671875, 0.7685546875, 0.2489013671875, 0.3330078125, 1.185546875, 0.1024169921875, 0.186767578125, -0.1534423828125, 0.4765625, 1.0166015625, -0.86767578125, -0.158447265625, 0.171630859375, 0.91064453125, 0.9248046875, 0.2890625, 1.578125, 0.242919921875, -0.73681640625, -0.31982421875, 0.50244140625, 0.1329345703125, -1.341796875, 0.431640625, 0.7890625, 1.2197265625, -0.80224609375, 0.2279052734375, -0.67626953125, 0.61279296875, -0.039581298828125, -1.2001953125, -0.615234375, 0.038177490234375, -1.490234375, 0.54296875, 2.498046875, 1.0498046875, 0.280517578125, 1.412109375, -0.207275390625, -1.4404296875, 0.5244140625, -0.55517578125, -0.5341796875, -0.47607421875, -1.0859375, 0.91796875, -0.634765625, 1.404296875, -0.333740234375, 1.0673828125, -0.443359375, 0.63671875, -0.10113525390625, -1.60546875, 1.0283203125, 0.308349609375, -0.75439453125, 0.60595703125, 1.55859375, -1.79296875, 1.533203125, -0.90478515625, 1.2392578125, 0.07440185546875, -0.347412109375, 0.79931640625, -0.0794677734375, -0.5673828125, -0.2281494140625, -0.293701171875, -0.451904296875, 1.064453125, -1.326171875, -0.60205078125, -0.615234375, -1.025390625, 0.062225341796875, -0.36181640625, -0.1387939453125, -0.11187744140625, -0.24169921875, -0.11083984375, 0.2081298828125, 0.2420654296875, 1.0595703125, -1.359375, -1.1650390625, -0.1446533203125, -0.53369140625, 0.51806640625, 2.28515625, 0.431396484375, 1.01953125, -0.89404296875, -0.397216796875, -0.434326171875, 0.466552734375, -0.54345703125, -1.1708984375, -0.68994140625, -0.9375, -0.05242919921875, -1.8466796875, -2.443359375, 0.2078857421875, -0.5986328125, 0.90673828125, -0.66015625, 0.083251953125, -0.64208984375, -0.489013671875, 0.67138671875, -0.039520263671875, 0.424072265625, -0.474853515625, -1.3955078125, -0.68505859375, -1.126953125, -1.5751953125, -0.271484375, 0.64794921875, 0.884765625, 1.904296875, -0.1566162109375, 0.765625, -0.43017578125, -0.32373046875, -1.1455078125, 0.62744140625, -0.51416015625, -0.92724609375, 0.03961181640625, -1.34765625, 0.00179290771484375, -1.140625, -0.0526123046875, 1.5146484375, 0.299072265625, -1.8310546875, 0.08447265625, -0.25439453125, -0.7861328125, -0.8935546875, -1.337890625, 0.64794921875, -0.045135498046875, -0.52294921875, 2.1015625, -1.1298828125, -0.90185546875, -0.75830078125, -0.0237884521484375, 3.634765625, 0.75634765625, -0.270263671875, -0.466796875, -0.2059326171875, 2.25, 0.806640625, 0.5673828125, 0.11822509765625, -0.70068359375, 0.7568359375, -0.353271484375, -0.0197296142578125, 1.541015625, 0.252197265625, 1.1611328125, 0.418212890625, -1.4658203125, 1.4111328125, -0.155029296875, 0.10003662109375, -0.26806640625, 0.546875, 0.2291259765625, 1.1396484375, -1.5029296875, 2.796875, 0.27197265625, 0.396728515625, 0.1134033203125, 1.4150390625, 0.75634765625, 0.457275390625, -0.96044921875, -0.125, 0.180419921875, -0.0184783935546875, -0.416015625, -0.8251953125, 0.7255859375, 0.416259765625, 0.61279296875, 0.8125, 1.0283203125, 0.71533203125, 0.1663818359375, -1.0595703125, 0.276611328125, 0.07635498046875, -0.51220703125, -0.2144775390625, -0.254638671875, 0.419677734375, -0.27392578125, 0.19384765625, 0.278076171875, -0.06378173828125, -0.47314453125, -0.27001953125, 1.73828125, -0.755859375, 0.9267578125, -0.1019287109375, 0.96630859375, -1.0927734375, 0.0002830028533935547, -1.0615234375, 0.501953125, -1.06640625, -0.323974609375, 0.325927734375, -0.461669921875, -2.197265625, 0.83251953125, 0.67626953125, -0.83447265625, 2.181640625, 1.033203125, -0.11859130859375, -0.416015625, 0.6845703125, -0.335693359375, 0.80078125, -0.43115234375, 0.54296875, 0.625, 1.705078125, -0.470703125, 0.306640625, -1.52734375, -0.52099609375, 0.88134765625, -0.3681640625, -0.390380859375, -0.164306640625, 0.0902099609375, -0.0882568359375, -1.802734375, 0.07135009765625, 0.41162109375, -0.44677734375, 0.6953125, 1.4921875, 0.45947265625, -0.2269287109375, -0.2161865234375, -0.61474609375, -0.324951171875, -0.487060546875, -1.392578125, 0.40478515625, -0.453125, 2.248046875, 0.822265625, 0.056243896484375, 1.1640625, -1.12890625, 0.44091796875, 0.23828125, -2.55859375, -0.330810546875, 0.91357421875, 0.76025390625, 0.34228515625, 2.27734375, -0.17138671875, 0.9052734375, -1.2626953125, -0.939453125, 1.603515625, -0.375244140625, 1.0703125, 0.06658935546875, 1.3583984375, 0.483642578125, 2.259765625, -0.50634765625, 0.92041015625, 1.7587890625, -0.46728515625, 0.081787109375, -0.54296875, 0.417724609375, -0.5478515625, -0.22265625, 0.7666015625, -0.4580078125, -0.30712890625, -0.07891845703125, -1.884765625, -0.127197265625, 0.8759765625, -0.68603515625, -1.375, 0.97802734375, 0.0902099609375, -0.72021484375, 0.90966796875, -0.347900390625, 0.054718017578125, 0.052734375, -0.1807861328125, 1.09765625, 0.923828125, 1.177734375, -0.26611328125, -0.73876953125, 0.33349609375, 1.4970703125, 0.65380859375, 0.287353515625, -0.798828125, 1.06640625, 0.6474609375, 0.1600341796875, -0.05120849609375, 1.2607421875, 0.34716796875, 0.2978515625, 0.229248046875, -0.619140625, 0.08062744140625, -1.322265625, -0.796875, 0.49169921875, 0.296630859375, 0.03106689453125, -0.8349609375, 0.75439453125, 1.3857421875, -0.8564453125, 1.201171875, 1.98046875, 0.77392578125, -1.8818359375, -0.66015625, 0.98583984375, 0.56005859375, 0.45947265625, -1.662109375, 1.8525390625, -0.52587890625, -0.389892578125, -0.458984375, 0.443359375, 0.150634765625, -0.884765625, 2.26171875, 0.587890625, 0.48974609375, -0.515625, 1.2783203125, 0.02447509765625, 0.41015625, 1.0166015625, -0.93505859375, 0.65087890625, -2.48046875, -0.96484375, -1.5185546875, -0.062469482421875, 0.275146484375, 0.611328125, 1.1201171875, 0.9189453125, -0.318359375, 0.0880126953125, -1.19140625, -1.0537109375, -0.66015625, 1.484375, 1.958984375, -0.4775390625, 1.0107421875, -0.5634765625, -0.8359375, 0.79150390625, 0.006694793701171875, 0.1619873046875, 1.009765625, 0.331298828125, 1.08203125, -0.199951171875, 0.54833984375, 0.689453125, -0.020355224609375, 0.11248779296875, -0.324462890625, -0.338623046875, -0.75390625, -0.8349609375, -0.60400390625, 0.055267333984375, 0.2314453125, -0.26953125, -1.259765625, -0.71484375, 0.11029052734375, -0.467041015625, -1.359375, -0.810546875, -1.4736328125, -0.8720703125, -0.86669921875, -0.8955078125, 0.69580078125, -0.86865234375, 0.447509765625, -0.84521484375, 0.556640625, -1.2294921875, -0.7734375, -1.345703125, 0.1688232421875, 0.214111328125, 1.0419921875, 0.346923828125, -0.2408447265625, -0.29345703125, 1.3330078125, 0.73046875, 0.321044921875, -0.6240234375, -0.057159423828125, 0.206787109375, 0.2119140625, 1.2431640625, -1.025390625, 1.04296875, 0.84423828125, -0.1317138671875, 0.400146484375, 1.390625, 0.177978515625, 0.74853515625, 1.1064453125, -1.3955078125, 2.41796875, 0.2110595703125, 0.5322265625, -0.86669921875, 0.53125, 1.0986328125, -0.9912109375, 0.09881591796875, -1.158203125, -1.1904296875, 0.303466796875, 0.321533203125, 0.99658203125, 1.2099609375, 0.12298583984375, -0.80419921875, 0.171630859375, 0.33544921875, -0.383544921875, -1.1650390625, 0.499755859375, -0.81494140625, -1.7919921875, 0.452392578125, 1.21484375, -0.342529296875, -0.2044677734375, -1.8408203125, -2.1171875, 0.458251953125, -1.37890625, 0.501953125, -0.036773681640625, 0.34521484375, -0.248779296875, -1.40625, -0.5859375, -1.0869140625, -3.181640625, 0.363037109375, -0.236083984375, -1.5205078125, -1.666015625, 0.2208251953125, 1.482421875, -0.058380126953125, 0.45166015625, -0.7158203125, -0.64794921875, -1.880859375, -0.4072265625, 0.049072265625, -0.66357421875, 1.865234375, 0.76220703125, -1.1455078125, 0.151123046875, 3.3984375, -1.1845703125, -0.583984375, -0.67236328125, 0.258544921875, -0.29736328125, 2.189453125, -0.6943359375, -0.242919921875, 1.30859375, 0.333740234375, -0.4169921875, -0.395263671875, -0.55810546875, -0.386962890625, 0.87548828125, 0.4951171875, -1.2158203125, -1.99609375, -1.1884765625, 0.387939453125, 1.224609375, 1.443359375, 1.2626953125, -0.058563232421875, 0.348876953125, -0.07110595703125, 0.07769775390625, -2.76953125, 0.45361328125, -0.0301971435546875, 0.34716796875, 1.37890625, -1.44921875, 0.3017578125, -0.9462890625, 0.402099609375, 1.3798828125, -1.912109375, 0.119873046875, -0.7646484375, -0.252685546875, -1.505859375, 0.70703125, 0.9404296875, -0.11370849609375, 0.1368408203125, 0.293701171875, -0.321533203125, 1.455078125, 0.65380859375, 1.3828125, -1.0283203125, -0.35986328125, 0.9619140625, 0.2333984375, -0.3603515625, 0.0022487640380859375, -2.283203125, 0.63916015625, -0.3427734375, -1.0390625, -0.462646484375, -0.24169921875, 0.52734375, -1.181640625, -0.7412109375, 0.0275115966796875, -2.076171875, -0.2017822265625, 1.3359375, -1.2275390625, -2.408203125, 0.5185546875, -0.146728515625, 0.344970703125, 0.4189453125, -0.6455078125, 0.2176513671875, 0.1871337890625, 1.0859375, -1.611328125, -0.08795166015625, 0.7255859375, 1.0029296875, -1.42578125, -0.54150390625, 0.43896484375, -0.335693359375, 1.5478515625, -0.6044921875, -0.64990234375, 0.273681640625, -1.755859375, -0.94580078125, 1.455078125, -0.685546875, -0.3955078125, 1.5205078125, 1.4150390625, -1.0712890625, 0.037689208984375, 1.2900390625, 1.525390625, -1.6435546875, 0.8203125, 0.53173828125, 1.603515625, -0.2607421875, -0.72998046875, -0.1767578125, -1.69140625, -0.22900390625, 0.307373046875, 0.759765625, 0.231689453125, 0.32861328125, -0.94580078125, 0.07720947265625, -1.4443359375, 0.491455078125, -0.32275390625, 1.2275390625, 0.59326171875, -1.9111328125, -0.08428955078125, -0.6337890625, 0.050506591796875, 0.160400390625, 0.70703125, 1.3203125, -0.65576171875, -0.47998046875, -0.5439453125, 0.103515625, -0.68359375, 1.345703125, -2.40234375, -0.775390625, -1.41796875, 1.484375, 0.2347412109375, 0.52978515625, 1.037109375, 0.0309295654296875, -0.970703125, -0.7919921875, -0.359375, 1.171875, -1.0810546875, 1.8203125, 0.322021484375, 1.4873046875, -0.82666015625, 0.70166015625, -1.3447265625, -0.113525390625, 0.90087890625, -1.00390625, 0.32666015625, 1.58203125, 0.5634765625, -0.619140625, -0.87255859375, -0.0270843505859375, -0.5068359375, 0.55859375, 1.2021484375, 1.4912109375, 1.85546875, 1.62109375, -0.55859375, -1.34765625, 0.3720703125, -1.2265625, 1.9482421875, 1.3525390625, -0.52685546875, 0.595703125, -0.265380859375, -1.123046875, 0.72021484375, -0.0050201416015625, -0.62109375, -1.826171875, -0.1895751953125, 0.1214599609375, 0.97119140625, -1.5478515625, 0.81494140625, -1.58984375, -0.7109375, -0.556640625, -1.1513671875, 1.931640625, 0.99462890625, 0.08734130859375, 0.49169921875, 1.6884765625, -1.0185546875, 0.609375, -0.1787109375, -0.4765625, 0.630859375, -0.67236328125, 0.0114898681640625, -0.677734375, -0.7509765625, -1.91796875, 1.724609375, 1.1572265625, -0.01824951171875, 0.24609375, 0.82763671875, -1.19921875, -0.513671875, 0.85302734375, 0.37158203125, -0.470458984375, -0.175537109375, 0.2022705078125, 0.347412109375, -1.2646484375, -0.3486328125, 0.587890625, 0.47509765625, -0.05120849609375, -0.015655517578125, 0.15966796875, -0.1748046875, 0.82958984375, 0.67578125, -0.060943603515625, 0.837890625, -1.45703125, 0.71142578125, 0.71533203125, -1.865234375, -0.10150146484375, -1.140625, 0.053009033203125, 1.3515625, 0.31298828125, 1.251953125, -0.451904296875, -1.1396484375, -1.6767578125, -0.630859375, 0.1993408203125, 3.17578125, -0.88916015625, -1.1904296875, -2.099609375, -1.7763671875, 0.446533203125, 1.21875, 1.6748046875, 0.22509765625, -0.350341796875, 0.278076171875, 0.76123046875, 0.256103515625, -0.66455078125, -0.68359375, -0.72216796875, 0.21484375, -1.716796875, -1.5224609375, 1.4755859375, -0.27001953125, 1.4150390625, 0.51416015625, 0.73583984375, 0.425537109375, -1.845703125, -0.50830078125, -0.56884765625, 0.59619140625, -1.2802734375, -0.1666259765625, 0.80615234375, -1.236328125, -1.7607421875, -0.11395263671875, -0.048095703125, 0.269287109375, -1.490234375, -2.009765625, 1.6748046875, 0.93017578125, 1.958984375, -0.1650390625, -1.1640625, 1.9482421875, -0.2174072265625, 1.23046875, 2.158203125, -1.2958984375, -2.41015625, -1.529296875, 1.875, -0.3896484375, -0.45556640625, 1.162109375, 0.56201171875, -0.021331787109375, -0.48486328125, -0.6923828125, 0.86474609375, -0.38671875, -1.046875, 0.02264404296875, -0.73974609375, -0.216552734375, 0.287841796875, -0.97998046875, -1.0654296875, -0.7177734375, 0.50341796875, 1.0302734375, 0.1627197265625, -0.927734375, -0.6953125, 0.7216796875, -0.441162109375, -0.02691650390625, -0.495361328125, 0.85595703125, -1.2119140625, -0.5966796875, -0.12066650390625, 0.340087890625, -0.80712890625, 1.19921875, -0.32666015625, 0.0670166015625, -0.82958984375, 0.311767578125, 0.0013790130615234375, -0.64208984375, -2.03515625, -0.84423828125, 0.2266845703125, 0.056854248046875, -0.76708984375, -0.51513671875, -0.13330078125, 0.5244140625, -0.48876953125, -0.5322265625, 2.298828125, -1.837890625, 0.086181640625, 0.399658203125, -0.30908203125, -0.74365234375, 0.328857421875, -0.61279296875, -0.65576171875, 0.07110595703125, -0.1456298828125, -1.318359375, 0.32421875, -0.50048828125, 0.81591796875, -0.71826171875, -0.62890625, -0.367919921875, -2.099609375, 0.10784912109375, 0.10736083984375, -0.69873046875, -2.228515625, 1.1806640625, -0.177734375, -0.71630859375, -0.392822265625, -0.08648681640625, -0.5869140625, -1.0810546875, 1.951171875, -1.3828125, -1.1787109375, -0.479736328125, -0.35205078125, -0.78173828125, 0.53515625, 0.361328125, 0.85693359375, 0.544921875, -1.1923828125, 0.156494140625, -0.84130859375, -1.0224609375, 0.97509765625, 0.1976318359375, -0.1519775390625, 1.5478515625, -0.14453125, -1.6552734375, -0.239501953125, 1.1982421875, 0.393798828125, -0.595703125, 1.1416015625, 1.0302734375, 1.2021484375, -1.9833984375, 0.318603515625, -1.5302734375, -0.9560546875, -0.39306640625, -0.320556640625, -0.056793212890625, -0.75634765625, -0.6474609375, 0.69482421875, -1.4013671875, -0.286865234375, 0.8349609375, 0.31787109375, 0.8330078125, 0.380859375, 0.74560546875, -0.310302734375, 0.1158447265625, 0.7705078125, 1.8125, -0.00946044921875, -0.98828125, -0.60400390625, -0.459716796875, 0.15576171875, 0.407958984375, -0.369140625, -0.404541015625, -1.701171875, 0.0706787109375, -1.2333984375, 0.355224609375, 0.19580078125, 0.07379150390625, -1.4892578125, 0.47998046875, -1.1572265625, 1.666015625, -3.02734375, -1.0283203125, -0.2493896484375, -2.23046875, -0.57763671875, 1.103515625, 0.280517578125, 0.74072265625, 0.91796875, -0.20263671875, -0.25341796875, 0.75927734375, 1.197265625, -0.468017578125, 1.4072265625, 0.53515625, 2.0078125, 1.6640625, 2.533203125, -0.484619140625, 0.08203125, 1.6533203125, 0.6953125, -0.8505859375, -2.0, -1.630859375, 2.521484375, -0.8134765625, 0.52734375, 1.7197265625, -0.7021484375, 1.1162109375, 0.351806640625, 0.473876953125, 0.283935546875, -1.2177734375, -0.6318359375, 0.52490234375, 0.384521484375, -0.2020263671875, -0.460205078125, 1.212890625, 0.445556640625, 1.7685546875, 0.284912109375, -0.40380859375, 0.417236328125, 0.4287109375, 0.61083984375, 1.708984375, 1.509765625, 0.34619140625, -1.09375, -0.779296875, 1.16796875, -1.0546875, -0.70751953125, 1.9169921875, 0.1478271484375, 0.50732421875, 0.460205078125, 0.609375, -0.323486328125, -0.226806640625, 1.9443359375, 0.630859375, -0.8017578125, 0.42919921875, 0.13330078125, 1.271484375, 3.0390625, -0.1544189453125, -0.7314453125, -2.0234375, -0.28662109375, -0.08123779296875, 0.93798828125, 0.332275390625, 1.216796875, -0.01053619384765625, -0.019622802734375, 0.81884765625, 0.63134765625, -1.130859375, -0.2115478515625, -1.3671875, -0.6572265625, -1.0576171875, 0.36474609375, 0.1036376953125, -0.8544921875, 0.85986328125, -1.501953125, -0.273681640625, -1.3642578125, -0.63525390625, -0.1878662109375, -1.0380859375, -0.96044921875, -1.150390625, 1.08203125, -1.1865234375, 0.755859375, -0.2432861328125, -0.07562255859375, -0.583984375, 0.310302734375, 0.35498046875, 0.76318359375, 0.015960693359375, 0.7734375, -0.3271484375, -1.1162109375, -0.342529296875, -0.85595703125, -0.08795166015625, -0.41455078125, 0.314697265625, 0.6533203125, 0.157470703125, 0.3857421875, -1.5078125, -0.1661376953125, 0.1873779296875, 1.71875, 0.051513671875, -0.5986328125, -1.017578125, -0.0081939697265625, -0.9697265625, -0.73388671875, 1.5927734375, 0.454833984375, 0.6728515625, -0.52783203125, -0.59130859375, -1.0400390625, 2.998046875, -1.3330078125, 0.2476806640625, -0.5712890625, 0.5615234375, -1.765625, 0.26904296875, -1.5439453125, -1.400390625, 0.04351806640625, 1.2255859375, 1.544921875, -0.27197265625, 0.12017822265625, -0.410888671875, -1.3525390625, 3.111328125, -1.2744140625, -1.22265625, 0.8994140625, -1.71484375, 0.3427734375, -0.465087890625, -1.748046875, -1.080078125, -1.9384765625, -1.294921875, 0.069091796875, 1.6552734375, -1.0986328125, -0.435791015625, 1.556640625, 1.6982421875, 1.1123046875, -0.5615234375, 0.29248046875, 1.283203125, 0.7578125, -1.482421875, 1.21875, -0.41796875, -0.3359375, -1.8994140625, 0.41650390625, -0.84033203125, -0.68603515625, -0.499755859375, 1.0458984375, -0.34130859375, 0.62841796875, 0.384521484375, 1.630859375, 0.08038330078125, -1.17578125, -0.08038330078125, 0.9697265625, -0.332763671875, -1.00390625, 1.35546875, 1.0263671875, -2.09375, -1.064453125, -0.345458984375, 1.68359375, 0.38720703125, 0.355224609375, -1.0556640625, -0.37890625, -0.392822265625, -1.611328125, 1.017578125, -0.56201171875, 0.6376953125, -0.48828125, -0.60302734375, -1.001953125, 1.0126953125, 0.0445556640625, 0.03460693359375, 0.97216796875, -1.2001953125, -0.62060546875, -1.32421875, -0.95654296875, 0.007579803466796875, 1.046875, 1.5888671875, -0.64990234375, -3.146484375, 0.66748046875, 0.8076171875, 0.496337890625, 0.72265625, -0.96142578125, 0.347412109375, -1.228515625, 0.407958984375, -2.916015625, -0.73681640625, -0.089599609375, -0.94677734375, -0.81298828125, -1.6708984375, 0.058990478515625, 0.407958984375, 1.04296875, -0.072021484375, -0.92431640625, -0.4423828125, -0.51318359375, -0.3720703125, -0.6337890625, -1.1142578125, 0.7568359375, -1.4248046875, 2.169921875, -0.40185546875, 0.685546875, 0.6474609375, -0.80224609375, 0.70361328125, 0.280517578125, 0.56103515625, 0.044219970703125, -0.033355712890625, -1.3837890625, 0.96875, 1.2568359375, -0.515625, 0.44140625, -2.291015625, -0.951171875, -0.90576171875, -0.113525390625, 0.54296875, 1.025390625, 0.2235107421875, 1.0263671875, -2.755859375, -0.467529296875, -0.26513671875, -0.3740234375, -0.62890625, 0.263427734375, -0.9072265625, -0.9833984375, 1.3759765625, 0.78662109375, -2.3671875, -1.9169921875, -0.68701171875, 1.921875, -0.759765625, -0.10693359375, -2.142578125, -0.84619140625, 1.3662109375, -1.7509765625, 0.76806640625, -1.6337890625, 0.365966796875, -2.083984375, 0.299072265625, 0.37841796875, -0.9853515625, -0.03521728515625, 0.448974609375, -0.990234375, 2.140625, 1.103515625, -0.65771484375, 1.3740234375, 0.248046875, 0.370849609375, 0.740234375, 1.28125, -0.31494140625, 0.34716796875, 0.468505859375, 0.1466064453125, -1.1708984375, -1.9208984375, 0.0809326171875, -1.08984375, 1.4443359375, -0.1741943359375, 0.1978759765625, -1.5888671875, -0.489501953125, -1.2666015625, 0.060150146484375, -0.330810546875, -0.94091796875, 0.96044921875, 0.26806640625, -0.1524658203125, 0.421630859375, 1.19921875, -1.1181640625, 0.61181640625, 1.2734375, 2.65234375, 0.49169921875, -0.63623046875, -1.4365234375, -0.3212890625, 0.46923828125, -0.406005859375, -1.1123046875, 0.4912109375, -0.050140380859375, 1.8828125, -1.09765625, -1.4765625, 0.4951171875, -1.3427734375, 0.58056640625, 0.27734375, 0.7431640625, 1.3193359375, 1.37109375, 0.958984375, 0.2822265625, -0.5634765625, -0.658203125, -0.1572265625, -0.85693359375, -0.16357421875, -0.61865234375, -0.587890625, 1.3388671875, -0.09735107421875, 0.263916015625, 0.83349609375, -0.56787109375, 0.224853515625, -1.34375, 0.44482421875, -0.339111328125, 1.2470703125, -0.07904052734375, -0.0855712890625, -1.404296875, 0.6328125, 0.20458984375, 0.440673828125, -1.6455078125, -0.765625, -0.80224609375, -0.06036376953125, -0.1990966796875, -0.488037109375, 0.67626953125, 1.1083984375, 0.336181640625, -0.341552734375, -0.6513671875, 0.473388671875, 0.56201171875, 0.5986328125, 0.1968994140625, -0.35107421875, -0.5322265625, -0.1837158203125, 0.39501953125, 2.01953125, -1.734375, 0.7744140625, -0.57763671875, 0.53955078125, -0.4208984375, -0.14501953125, 0.00024700164794921875, -1.0654296875, 0.892578125, -0.2161865234375, 0.0831298828125, -0.29248046875, -0.86376953125, -0.051544189453125, 1.1572265625, 0.7041015625, 0.104248046875, -1.64453125, -0.58349609375, -0.88427734375, -0.2154541015625, 0.775390625, -1.6298828125, -1.3193359375, -0.60888671875, -0.177001953125, -0.51513671875, -1.232421875, -0.59033203125, -0.09918212890625, 1.0302734375, 1.45703125, 0.37841796875, 1.2802734375, 0.50048828125, -0.0562744140625, -0.211669921875, -1.2724609375, 1.072265625, 0.108154296875, 0.305908203125, 0.486083984375, 0.69189453125, -0.295166015625, -0.52978515625, 0.16455078125, 0.830078125, 0.454345703125, 0.51123046875, 2.1328125, -0.302490234375, -0.2254638671875, 0.9423828125, -0.3974609375, 0.82080078125, -0.07415771484375, -1.0361328125, 0.916015625, -0.7890625, 0.1533203125, 0.024627685546875, 1.244140625, -0.552734375, -0.8818359375, -0.468017578125, 0.75830078125, 0.44970703125, -1.2626953125, -0.1751708984375, -0.68017578125, 0.9375, 1.587890625, -0.45751953125, -0.09967041015625, -0.54248046875, -0.1876220703125, -0.90185546875, 2.00390625, -0.01605224609375, 0.51904296875, -0.572265625, -1.1923828125, -1.701171875, -1.8203125, 0.90576171875, 0.78662109375, 0.438720703125, -0.438720703125, -0.84814453125, 0.1641845703125, 1.431640625, 0.76416015625, -1.34375, 0.6923828125, -1.1083984375, -0.8115234375, 0.60107421875, -0.478515625, -0.11126708984375, -2.193359375, 0.96728515625, 0.5576171875, 0.595703125, 0.54443359375, -0.145263671875, 1.25, 0.52001953125, 0.50146484375, -0.53125, 0.337646484375, -1.24609375, 0.31982421875, -1.099609375, -0.95166015625, -2.177734375, 1.47265625, -0.5126953125, -1.3896484375, 0.049224853515625, -0.4384765625, -0.6181640625, -0.033447265625, -1.025390625, 0.99072265625, -1.5078125, 0.78271484375, 1.556640625, -0.5654296875, 0.818359375, 0.32763671875, -0.365234375, 0.0268707275390625, 0.41357421875, -0.57861328125, -0.619140625, 1.783203125, 0.60986328125, 0.287353515625, -0.8271484375, -0.013763427734375, -1.23828125, -1.3701171875, -0.5146484375, -1.21484375, -0.1396484375, 0.02618408203125, -1.1416015625, -0.139404296875, -0.34326171875, 0.685546875, -0.310791015625, -0.264404296875, -1.6044921875, 0.80078125, 1.623046875, 2.072265625, 0.307861328125, 1.8720703125, 0.7841796875, 0.77490234375, 1.255859375, -0.194091796875, -0.34716796875, -0.35693359375, 0.00018405914306640625, 1.177734375, -0.259521484375, -0.3505859375, -0.71337890625, 0.177001953125, 1.015625, -1.2421875, -1.4404296875, -0.7939453125, -0.54833984375, -0.0106201171875, 0.58544921875, 1.5, 1.126953125, -0.240234375, -0.21142578125, 0.14794921875, 0.283447265625, -2.849609375, -1.7685546875, -1.7275390625, 0.159912109375, 1.0498046875, 1.064453125, -0.8642578125, -0.2489013671875, 0.12255859375, -1.51953125, 0.71630859375, -0.57861328125, 0.26806640625, -1.833984375, -0.31591796875, -0.9052734375, -0.841796875, 1.501953125, -0.4580078125, 1.0263671875, -0.423095703125, 1.49609375, 0.78857421875, -1.015625, 0.94775390625, -0.0950927734375, -0.0762939453125, 1.234375, 0.1968994140625, -0.9306640625, 0.12432861328125, 1.447265625, -0.11273193359375, -0.734375, -1.4375, -1.080078125, -1.1318359375, 1.373046875, -0.2200927734375, -9.202957153320312e-05, -0.89599609375, 1.5966796875, 0.440673828125, -1.3330078125, 1.2666015625, -1.005859375, -0.1441650390625, 0.568359375, 1.3857421875, -2.546875, 0.85546875, 0.1961669921875, -0.98291015625, -0.31298828125, 0.611328125, -0.053436279296875, -1.95703125, 0.04351806640625, -1.373046875, 0.2091064453125, -0.6826171875, 0.272705078125, 1.5859375, 0.8193359375, 1.158203125, -0.06048583984375, -0.366943359375, -1.1796875, 1.6318359375, -0.80810546875, 0.237548828125, -0.2449951171875, -0.1517333984375, -1.7900390625, -1.2587890625, -1.265625, 2.48046875, -0.89111328125, 0.5048828125, -1.576171875, -1.02734375, 1.1083984375, 0.372802734375, -1.1064453125, 0.036895751953125, -0.131103515625, 1.3984375, -1.119140625, 0.406494140625, 1.1376953125, 0.26416015625, -0.71044921875, 1.236328125, -0.9091796875, -0.92333984375, -0.1427001953125, -1.388671875, 0.218505859375, -1.037109375, -1.9326171875, -0.6357421875, 0.05706787109375, 0.5712890625, 1.23828125, -0.317138671875, -0.802734375, -1.470703125, -0.295166015625, 0.361083984375, -0.40185546875, -0.2266845703125, -1.474609375, -0.3984375, -0.490234375, 1.4443359375, 0.63916015625, 0.70068359375, -0.880859375, 0.7109375, 0.346435546875, -1.896484375, 1.009765625, 1.3876953125, 0.270263671875, -0.06512451171875, -0.054656982421875, 0.1646728515625, -0.3916015625, -0.70068359375, 0.77978515625, -0.64404296875, 0.6357421875, -0.5947265625, 0.377197265625, 1.20703125, -0.69921875, -0.249755859375, 1.4013671875, 1.318359375, 0.053680419921875, 1.1982421875, 0.36767578125, 2.3046875, 1.3115234375, 0.5478515625, 0.145751953125, 0.51611328125, 0.17041015625, 1.3349609375, 0.22216796875, 0.51123046875, -0.962890625, -0.09820556640625, -0.802734375, -0.0372314453125, 2.482421875, 0.5625, -0.72705078125, 0.74560546875, -0.2252197265625, 1.0791015625, 0.58203125, -1.166015625, -0.2374267578125, -0.53173828125, 0.338623046875, 1.890625, 0.2353515625, -1.947265625, 0.98828125, 0.283935546875, -0.94921875, -1.1181640625, -0.51171875, 0.7744140625, 0.453857421875, -1.150390625, 0.50634765625, 0.384521484375, 0.7177734375, -1.13671875, -0.424072265625, -0.69384765625, -1.017578125, -1.0634765625, 0.035797119140625, 1.767578125, -0.77490234375, -1.0634765625, -0.495849609375, 1.5576171875, -0.90869140625, -0.1776123046875, -2.095703125, -0.2060546875, -1.13671875, 1.396484375, -0.85693359375, -0.259033203125, 1.6357421875, -0.451416015625, -0.748046875, 0.058746337890625, 2.5234375, 2.041015625, -0.6923828125, 0.427490234375, -1.5732421875, -1.576171875, 0.88232421875, -0.43017578125, 1.1767578125, -0.1380615234375, 0.334716796875, -1.341796875, 0.325927734375, -0.3798828125, -0.90185546875, 1.4345703125, 0.33984375, -0.65283203125, -0.6044921875, 0.257080078125, -0.481689453125, 0.438720703125, -0.58642578125, 0.256103515625, -0.97802734375, 0.64453125, 0.280517578125, -0.241455078125, -0.487548828125, 0.83984375, -0.04742431640625, -1.2119140625, -0.0303192138671875, -1.8974609375, -0.329833984375, 1.640625, -0.52099609375, -0.76611328125, -0.447998046875, 1.578125, 1.466796875, -0.391845703125, -1.2802734375, -0.9609375, 0.29638671875, 0.48779296875, -0.051605224609375, 0.327880859375, -1.58203125, 0.8818359375, 0.42236328125, -0.05615234375, -1.80078125, 0.1319580078125, 0.54541015625, 0.378173828125, 0.94677734375, 1.25390625, 1.6630859375, 1.87890625, 0.82177734375, -1.7177734375, -1.5625, -0.51611328125, 2.482421875, 0.70849609375, -0.1455078125, 1.818359375, 2.080078125, 0.48974609375, 0.1845703125, -1.111328125, 1.2412109375, -1.43359375, 1.974609375, -0.57470703125, -0.5751953125, 0.489013671875, 0.09979248046875, 1.916015625, -1.2255859375, -1.47265625, -1.0400390625, -0.974609375, -1.30078125, 1.462890625, -0.7333984375, 0.8173828125, 0.814453125, 1.513671875, -0.09967041015625, 1.1083984375, -0.71435546875, -0.90283203125, -0.297119140625, 1.01953125, -0.99755859375, -1.27734375, 0.228271484375, -0.775390625, -0.2098388671875, 0.65966796875, 0.315673828125, -0.66015625, -0.09014892578125, 0.0159912109375, 1.1748046875, -0.8701171875, 1.255859375, -0.57666015625, 0.9736328125, -0.8388671875, -0.30859375, -1.04296875, -0.74560546875, -0.6962890625, 0.09918212890625, -0.43701171875, -2.138671875, -1.3134765625, -0.82763671875, 0.4248046875, 0.2176513671875, -0.09173583984375, 1.0478515625, -0.93896484375, 1.2255859375, 1.1865234375, 0.75341796875, -0.2353515625, -1.4189453125, 1.138671875, 1.0341796875, -0.61376953125, 0.70458984375, 2.21875, 0.60107421875, 0.7138671875, -0.335693359375, 0.927734375, 0.252685546875, -1.0654296875, -0.4228515625, -0.040130615234375, 0.343505859375, -0.00977325439453125, -0.149169921875, 0.52783203125, -1.341796875, -0.7529296875, 0.66162109375, 1.197265625, 0.348388671875, -0.34375, 0.08612060546875, -0.830078125, 1.5146484375, -0.96142578125, -0.91748046875, 0.6123046875, 0.289794921875, 0.63037109375, 0.8447265625, 0.77392578125, 0.55078125, 0.568359375, 1.1005859375, -0.426513671875, -2.671875, 1.4853515625, -1.1875, 0.77197265625, 0.429443359375, -0.578125, -1.3876953125, -1.234375, -0.487060546875, 0.6572265625, 0.12030029296875, -0.65771484375, 0.072021484375, 1.7236328125, 0.228759765625, 0.2442626953125, -0.9521484375, -0.286865234375, 0.64013671875, -1.1611328125, -0.7734375, 0.162841796875, -0.351318359375, -0.164306640625, 0.61572265625, 1.21484375, 0.27685546875, -1.22265625, -0.4462890625, -0.343994140625, 0.2412109375, -0.18701171875, -0.12432861328125, 0.222900390625, -0.12042236328125, -1.5634765625, -0.1524658203125, 0.419921875, -1.75, -0.22216796875, -0.38720703125, -0.83447265625, -0.02435302734375, -0.074951171875, 0.83251953125, 0.1700439453125, -0.254150390625, 0.20849609375, -0.1817626953125, 0.415283203125, 0.6875, 0.494873046875, 0.8994140625, 0.0243377685546875, -0.1578369140625, 0.78466796875, 0.4453125, -0.0955810546875, -1.4833984375, 1.2333984375, 3.1640625, -1.6875, -2.048828125, 0.79150390625, 1.51953125, -1.6513671875, 0.7373046875, -0.82373046875, 1.5244140625, -0.00995635986328125, -0.228515625, -1.1533203125, -0.74755859375, -0.466552734375, 1.4423828125, -0.42529296875, -0.60546875, 1.0888671875, 0.281494140625, 1.28515625, -0.61083984375, 0.460693359375, -0.94970703125, -0.77197265625, 0.2763671875, -0.802734375, 0.501953125, -0.2216796875, -0.3779296875, -1.4423828125, 0.99169921875, -0.35791015625, 1.32421875, 1.2001953125, 0.77294921875, -1.0380859375, 0.381103515625, 0.89501953125, -2.478515625, -0.45166015625, -0.68359375, -0.455322265625, 1.2216796875, -0.32080078125, 0.0885009765625, 0.4697265625, -1.216796875, 1.2294921875, 0.043060302734375, -0.056671142578125, -0.60009765625, 0.01503753662109375, -1.09375, -0.779296875, 0.41796875, -2.109375, -0.0280914306640625, -1.15625, 1.408203125, 0.044830322265625, 0.11279296875, -1.0361328125, 0.32666015625, 0.1751708984375, -0.400146484375, 0.2452392578125, 0.437255859375, 0.650390625, 0.07342529296875, 0.4541015625, 1.0859375, -0.75146484375, -0.6142578125, -0.2056884765625, 0.388916015625, 0.5498046875, 0.1522216796875, 1.744140625, 1.416015625, -0.29638671875, 1.2705078125, -0.54931640625, 0.07177734375, -0.317138671875, 0.2666015625, 0.8779296875, 0.0518798828125, 0.097900390625, 0.318603515625, -0.88427734375, -1.072265625, -1.4052734375, 0.7177734375, 0.305908203125, 1.2900390625, -0.39599609375, -2.2890625, 1.677734375, -2.19921875, 0.1251220703125, 0.498779296875, -0.1839599609375, -0.16650390625, -0.390869140625, 0.6259765625, -0.1273193359375, 0.2159423828125, -0.01526641845703125, 0.54833984375, -1.3583984375, -0.794921875, -0.53759765625, 0.94921875, -1.431640625, 0.5546875, 0.87158203125, 0.564453125, -0.47314453125, 0.9130859375, -1.232421875, 1.4208984375, -1.080078125, -0.27197265625, 1.0732421875, -0.0303192138671875, 0.038665771484375, -0.92333984375, -1.515625, 1.1005859375, 1.060546875, 0.67919921875, 1.28515625, 1.8798828125, -0.1591796875, -0.76708984375, 0.164794921875, 0.255126953125, -0.466796875, -0.669921875, 0.2445068359375, -0.2476806640625, -1.048828125, 1.8134765625, 1.6083984375, -0.1922607421875, -0.67041015625, 0.3671875, -0.400634765625, -0.62548828125, 0.57958984375, 0.1962890625, 1.3056640625, 0.134765625, 0.75634765625, -0.374755859375, -0.23681640625, -0.97021484375, 1.5888671875, 0.73486328125, -0.1024169921875, -0.346923828125, -1.123046875, 0.119873046875, -0.20458984375, 0.3974609375, 1.6572265625, 0.77880859375, -1.3369140625, 1.62109375, -1.0498046875, 0.10223388671875, 1.51171875, 1.77734375, -0.42431640625, 1.0244140625, 1.4423828125, -0.0704345703125, -0.225341796875, 1.1025390625, -2.037109375, -0.114990234375, -0.4873046875, -0.62060546875, 0.2354736328125, -0.85546875, 0.3662109375, -0.22412109375, 0.9541015625, -0.81591796875, -0.95361328125, -0.38427734375, -0.34423828125, -0.132080078125, -0.313720703125, 0.8759765625, -1.6181640625, 2.560546875, 0.5810546875, -1.2333984375, -0.62890625, -1.078125, 0.6025390625, 0.25341796875, 0.83056640625, -1.185546875, 0.2347412109375, -0.417724609375, -0.193359375, -1.2236328125, -1.939453125, -0.5361328125, -1.3046875, -0.72314453125, -0.97119140625, 0.939453125, 1.37109375, -0.305908203125, -0.76123046875, -0.11767578125, 1.0712890625, -0.212890625, 0.2587890625, -0.054046630859375, -0.439453125, -1.2568359375, 0.1444091796875, 1.0419921875, -0.0069732666015625, -0.7470703125, -0.6240234375, 1.947265625, -0.1796875, 0.5244140625, 0.04608154296875, -0.52978515625, -0.42822265625, 0.8564453125, -1.146484375, 0.6259765625, -2.20703125, 0.8740234375, 0.1456298828125, 0.368896484375, 0.35400390625, -0.1651611328125, 0.76513671875, -0.9228515625, 0.360595703125, 0.7607421875, 0.09967041015625, 1.44140625, 0.0714111328125, -0.208984375, -1.3173828125, 1.228515625, -1.328125, 1.15625, 1.7431640625, 1.2783203125, 0.0129241943359375, -0.3330078125, 0.65771484375, -0.001445770263671875, 1.142578125, 0.03289794921875, -1.0048828125, 1.0595703125, 0.451416015625, 0.2459716796875, 0.07635498046875, 0.469970703125, 0.1656494140625, 0.54052734375, 1.158203125, 0.0225372314453125, -1.3291015625, 1.044921875, 2.169921875, -0.0186004638671875, -0.434326171875, 0.40625, -0.5908203125, -0.056121826171875, -0.3818359375, 0.411376953125, -0.1077880859375, 0.5107421875, -0.3671875, -0.0240325927734375, 2.279296875, 1.0595703125, 0.295654296875, 1.0810546875, 0.97412109375, -0.06060791015625, -0.69384765625, -0.646484375, 0.533203125, -1.4892578125, 0.5380859375, -0.09405517578125, 2.162109375, 1.3095703125, -0.1807861328125, -0.459228515625, 0.3779296875, 0.64306640625, -0.15234375, 1.2744140625, 0.583984375, 0.00562286376953125, -0.70556640625, 1.2275390625, -0.89013671875, -0.8466796875, -0.13330078125, 0.54736328125, 1.51171875, -0.44189453125, -0.2978515625, -1.2587890625, -1.60546875, 0.3154296875, 1.154296875, 0.56884765625, 0.8154296875, -1.2685546875, -0.321044921875, 0.099609375, -1.2890625, 1.052734375, -1.130859375, -0.9033203125, -0.92578125, -1.8056640625, 0.459716796875, -1.1943359375, 0.52685546875, 0.43310546875, -1.6865234375, 0.1160888671875, 1.181640625, -0.55859375, -1.3681640625, 0.404296875, 1.5927734375, 0.25537109375, 1.5087890625, -0.95654296875, 0.7109375, 0.1732177734375, 0.213134765625, 1.3203125, -0.2001953125, -0.880859375, 2.224609375, 0.84765625, -1.0205078125, 1.1943359375, 0.46337890625, 0.7978515625, -0.1390380859375, -0.64306640625, 0.4501953125, 1.12109375, -1.0615234375, -0.97802734375, 0.66796875, 1.6162109375, 2.052734375, -2.732421875, 0.71826171875, -0.2320556640625, 0.052642822265625, -0.73583984375, -1.9677734375, -0.2666015625, 0.143310546875, -0.65576171875, -1.689453125, 0.305419921875, 0.369140625, -0.1319580078125, 1.6484375, -1.603515625, -0.22607421875, 0.430908203125, -0.93994140625, 0.60595703125, -0.7333984375, -0.412353515625, 0.039947509765625, -0.9619140625, 0.50634765625, -0.3076171875, -1.50390625, -0.19189453125, 1.65625, 0.630859375, -0.068115234375, -0.6357421875, -0.76318359375, 0.2408447265625, -0.360595703125, -0.01763916015625, 0.7275390625, -0.2445068359375, -0.48046875, 1.1123046875, -2.55078125, 1.4306640625, 0.5556640625, 0.767578125, -0.330078125, 0.55322265625, 1.2470703125, -0.89013671875, -1.205078125, -0.375244140625, 0.6044921875, -0.03497314453125, 1.0458984375, 1.587890625, 1.6279296875, -0.08203125, 0.14794921875, -0.419677734375, 0.70458984375, -0.283447265625, 0.646484375, -1.00390625, -0.9248046875, 1.8818359375, -1.31640625, -2.791015625, -1.623046875, -1.0009765625, 1.56640625, 0.75439453125, 0.41552734375, 0.072021484375, 1.3876953125, -0.9833984375, 0.054412841796875, -1.2578125, -1.9697265625, 0.078369140625, 0.1690673828125, 0.427001953125, 0.5068359375, -1.3193359375, -0.2325439453125, -0.57861328125, 0.11029052734375, 1.546875, -0.98388671875, 0.60546875, 0.40673828125, -0.0999755859375, -1.1513671875, 0.461669921875, -0.81787109375, 1.24609375, -1.2841796875, -1.3759765625, -0.42041015625, 0.25537109375, 2.189453125, 1.0751953125, -1.802734375, -0.0982666015625, -0.5771484375, 0.299072265625, 1.634765625, -0.46435546875, 0.5205078125, -0.59765625, -1.5732421875, -0.8408203125, 0.5283203125, -0.888671875, 0.572265625, -0.199951171875, -2.3828125, -0.314208984375, -0.5439453125, -2.763671875, -0.92626953125, 2.107421875, 1.02734375, 1.638671875, 0.1453857421875, 1.427734375, 0.473876953125, -1.5615234375, -0.1944580078125, 0.58837890625, -0.3671875, -0.736328125, -0.98779296875, 1.11328125, 0.93701171875, 1.0927734375, 0.5546875, 0.83251953125, -0.1478271484375, -0.7236328125, 1.1044921875, 1.423828125, -0.54345703125, 0.165283203125, 0.1845703125, -0.248779296875, 1.41015625, -1.5380859375, -0.76904296875, 1.9775390625, -0.79248046875, -0.4033203125, 1.4072265625, -0.48095703125, 0.6826171875, 1.5107421875, 1.265625, -1.1943359375, -0.403076171875, -0.640625, -0.025360107421875, 0.9697265625, -1.419921875, 0.58154296875, 1.74609375, 1.0615234375, -0.013641357421875, -2.076171875, 0.349609375, 1.4716796875, -1.19921875, 0.84716796875, -0.362060546875, -0.11016845703125, 0.293701171875, 0.352783203125, -0.59033203125, -1.46484375, -0.441650390625, 0.177734375, 0.49853515625, -0.5703125, 0.136962890625, 0.32861328125, 0.3828125, 1.5859375, 0.4326171875, -1.1875, 1.396484375, -0.21826171875, 1.1142578125, 0.1395263671875, 0.04400634765625, 0.25244140625, 1.2001953125, 0.340576171875, 0.201171875, 0.900390625, 0.1593017578125, 0.79833984375, -0.9521484375, -1.2841796875, -0.293212890625, -1.9267578125, -0.88623046875, 1.390625, -0.55419921875, -0.56103515625, -1.0537109375, 1.4150390625, -0.73828125, -0.034088134765625, -0.1883544921875, 0.287353515625, 2.103515625, 0.436279296875, 0.485595703125, -0.10089111328125, -0.364501953125, -1.2421875, -0.619140625, -0.33154296875, -0.151611328125, 0.55517578125, 0.50244140625, 0.47412109375, -0.611328125, -0.2027587890625, -0.2091064453125, -1.4599609375, 1.265625, 1.3134765625, -1.193359375, 1.0908203125, 0.0292205810546875, 0.79931640625, 0.22412109375, -0.36279296875, -1.615234375, 0.464599609375, 1.1572265625, -0.310302734375, -0.7578125, 2.046875, 0.88232421875, -0.331787109375, -0.27001953125, 2.140625, 0.59619140625, 1.044921875, 0.2099609375, 1.216796875, -3.09765625, 0.7509765625, -1.654296875, 0.6181640625, 0.57861328125, 0.69189453125, 0.192138671875, -0.8876953125, 0.76171875, -1.0947265625, 0.323974609375, -1.5732421875, -0.1295166015625, 0.0103912353515625, -1.716796875, -0.447021484375, -0.466552734375, -0.033050537109375, 0.34765625, 0.96826171875, 0.316650390625, -1.76171875, -0.439208984375, -0.56005859375, 1.5908203125, -1.18359375, -1.3388671875, 0.337646484375, 0.334716796875, 0.455810546875, -1.3759765625, 0.27197265625, 1.55078125, -0.264404296875, 1.5693359375, -0.9375, -0.3017578125, -1.7353515625, -0.419677734375, -0.71923828125, 0.34130859375, -0.654296875, 0.5751953125, 1.986328125, -0.412353515625, -0.3837890625, 0.438720703125, 0.2568359375, 1.6083984375, -0.1600341796875, 0.155517578125, 1.7412109375, 1.103515625, -0.1614990234375, 0.26220703125, 0.1810302734375, 0.0283660888671875, -2.125, -0.79736328125, 1.1728515625, -0.65869140625, 0.304443359375, -0.456298828125, -0.228759765625, 0.7001953125, 2.845703125, -0.350341796875, 0.251220703125, -0.194580078125, 0.78125, 2.541015625, -0.51318359375, -0.45263671875, -1.6748046875, 0.1025390625, -0.78662109375, 0.413330078125, -0.09869384765625, -0.452880859375, 0.35693359375, -1.1279296875, 1.9130859375, -0.243408203125, 1.119140625, 0.287353515625, -0.8603515625, -1.1513671875, 0.97998046875, 0.477294921875, -0.414794921875, 1.138671875, 1.150390625, -1.4775390625, -0.6015625, -0.6630859375, 2.111328125, -0.388671875, 1.380859375, -0.135009765625, 0.65966796875, -0.8515625, -0.8349609375, 2.349609375, -0.615234375, -0.1934814453125, -0.263916015625, -0.99365234375, 0.51806640625, -0.11553955078125, -0.833984375, -0.78173828125, -1.9189453125, -0.12548828125, -2.78515625, 2.25390625, 0.568359375, 1.0205078125, -2.009765625, 1.59765625, 1.19921875, -0.689453125, -1.0986328125, -0.63330078125, 1.57421875, -1.0205078125, 0.0450439453125, 0.064453125, 0.314208984375, -0.255126953125, 1.5751953125, -1.1689453125, 0.1981201171875, -1.736328125, 0.97802734375, 0.88037109375, 0.85009765625, -0.55810546875, 0.85791015625, -0.52197265625, -1.5283203125, -0.66845703125, -0.07086181640625, 0.17578125, -0.194091796875, 0.0634765625, -0.230712890625, -1.7978515625, 0.02203369140625, 0.95947265625, -1.8154296875, -0.05108642578125, -0.008392333984375, -0.7548828125, -0.407958984375, -0.1258544921875, 0.336669921875, 1.291015625, -0.8583984375, 0.31640625, -1.3525390625, -0.047637939453125, -2.0625, -0.66162109375, -0.7265625, -0.65576171875, 0.35205078125, -1.7744140625, 0.541015625, -0.78564453125, -0.30859375, -1.98046875, 0.073486328125, -1.52734375, -0.0992431640625, 0.119384765625, -1.5146484375, -0.23974609375, 1.5458984375, -0.50439453125, 0.150634765625, -0.1715087890625, 0.11260986328125, -0.22314453125, -0.054412841796875, 0.1434326171875, 0.047119140625, -0.50927734375, 0.1822509765625, 0.12158203125, -0.347412109375, 0.99462890625, 0.1915283203125, 0.487060546875, 1.8857421875, 0.1826171875, 0.73046875, 1.4658203125, -0.252197265625, -0.474609375, 1.0234375, 0.83984375, 0.17822265625, -2.125, 1.4609375, -1.400390625, 0.89501953125, -0.2371826171875, 0.8828125, 0.875, -3.263671875, 2.583984375, 1.08203125, 0.63916015625, 0.478271484375, 0.65966796875, 1.248046875, 1.025390625, -1.9443359375, -1.0673828125, -0.4609375, 0.77197265625, -1.052734375, 1.6728515625, -0.361328125, -0.626953125, -2.38671875, 1.24609375, -0.6201171875, 1.64453125, -0.1676025390625, -1.3310546875, 0.8720703125, -0.6806640625, 0.75048828125, -0.150146484375, -0.3232421875, 2.892578125, 0.38916015625, -1.8603515625, 0.286865234375, -1.8974609375, 1.5810546875, 1.1923828125, -0.19921875, 0.72216796875, -0.8193359375, -0.88427734375, -0.43310546875, -0.279541015625, -0.1427001953125, 0.168212890625, 0.45458984375, 0.888671875, -0.65771484375, 0.97119140625, -0.140380859375, -2.765625, -1.9326171875, -0.2054443359375, 1.072265625, 1.5830078125, -0.08355712890625, 0.365478515625, -0.145263671875, 1.2529296875, -2.541015625, -2.94140625, 1.103515625, 0.1627197265625, 1.27734375, 0.410888671875, 1.369140625, -0.381591796875, 1.62109375, -0.8203125, -0.6181640625, 0.97900390625, 0.2020263671875, -0.6103515625, 0.81396484375, 0.73828125, 1.91015625, 0.66259765625, -3.130859375, 0.2021484375, 0.3115234375, -0.8701171875, -1.66015625, 2.173828125, -1.466796875, 0.64404296875, 0.383544921875, 0.62548828125, -1.7646484375, 0.005413055419921875, -0.78125, -0.352294921875, -1.9951171875, 1.671875, -0.39404296875, 0.332275390625, 0.5537109375, 0.5107421875, 0.235595703125, 0.2467041015625, -1.1865234375, 0.591796875, 0.59228515625, 0.82958984375, 2.619140625, 0.6748046875, 0.7255859375, 0.3125, -1.69921875, -0.365966796875, 1.0830078125, -0.78955078125, 0.105224609375, -1.267578125, -2.140625, 0.703125, 2.56640625, -0.78759765625, 0.226318359375, 0.97802734375, 0.5888671875, 0.591796875, -0.054412841796875, 1.720703125, 0.46875, 0.32275390625, -0.41796875, -0.0977783203125, 0.435546875, -0.5771484375, -0.609375, -1.0537109375, 1.484375, 1.359375, -0.953125, -0.7353515625, -1.2783203125, -0.8486328125, -0.068359375, 1.9326171875, 1.3798828125, -2.267578125, -1.0234375, 0.64404296875, 0.75634765625, -0.432373046875, 0.016693115234375, -0.5830078125, 1.720703125, -1.3359375, -0.02447509765625, -1.3759765625, -0.50390625, 0.22412109375, 0.406982421875, -0.45263671875, -0.317626953125, -2.33203125, -0.0673828125, -1.19140625, -0.54541015625, 1.5537109375, -1.4775390625, -0.5380859375, 0.72314453125, 0.44580078125, 1.7119140625, -1.001953125, -0.10546875, 1.1669921875, -0.55517578125, 0.486572265625, 1.3115234375, 1.3564453125, -0.51513671875, 0.6982421875, 3.169921875, 0.62255859375, -0.35400390625, 0.41943359375, -1.46484375, 0.0239715576171875, 0.206787109375, -0.607421875, 1.6015625, 0.10577392578125, -0.8759765625, -2.2421875, 0.58056640625, 1.025390625, 0.275146484375, 0.6875, 0.82421875, -0.033843994140625, -0.239501953125, -1.0205078125, -0.57568359375, -1.2841796875, -0.650390625, 0.38916015625, 1.548828125, 0.42529296875, -0.0004181861877441406, -2.044921875, -0.20654296875, 0.1219482421875, 0.4404296875, -0.2421875, 0.04833984375, 0.86279296875, 0.59521484375, -0.78369140625, -1.0322265625, 0.9375, 0.66162109375, 1.2421875, 1.5361328125, 1.34375, 0.88232421875, 0.9267578125, -0.487060546875, 1.015625, 0.99951171875, 0.642578125, 0.77734375, -0.448974609375, -0.7080078125, 2.060546875, 0.71630859375, 0.61279296875, -0.7197265625, 0.53515625, 0.0416259765625, -0.1832275390625, -1.1494140625, 1.58984375, -0.1168212890625, -0.40625, -2.595703125, 0.2529296875, 1.4326171875, 1.5419921875, 0.01953125, 1.373046875, 0.440673828125, 2.216796875, -0.1044921875, -0.386474609375, -0.3505859375, 0.060302734375, 0.291015625, 0.338623046875, -0.0770263671875, -0.02032470703125, 0.1365966796875, 0.8740234375, 0.411865234375, 0.2734375, -0.859375, -0.0712890625, -0.08856201171875, 0.307861328125, 0.297607421875, 0.286376953125, 0.1414794921875, 1.49609375, -1.7958984375, 1.0537109375, -0.72314453125, -0.5419921875, 1.37109375, 0.20556640625, 0.52783203125, 1.3466796875, 0.1171875, -0.04705810546875, 1.0068359375, 0.16357421875, 0.385498046875, -1.142578125, -0.51611328125, -0.28515625, -0.53076171875, 0.39453125, 1.0615234375, 0.334228515625, 1.2705078125, 0.2384033203125, 0.94384765625, -0.58837890625, -0.47900390625, 0.7607421875, 0.91162109375, -0.31396484375, 0.87109375, -0.83203125, 1.4375, 1.2333984375, -0.73291015625, -0.59716796875, 2.0625, 0.083984375, 0.79833984375, -1.515625, 0.62060546875, -1.5380859375, -1.55078125, 1.7255859375, -0.85205078125, -1.1318359375, 0.7236328125, -1.07421875, -1.0439453125, 1.3837890625, 0.2015380859375, -0.82861328125, 0.086669921875, -0.0960693359375, 1.4052734375, 0.05413818359375, 0.2315673828125, -0.25, 0.6123046875, 0.01107025146484375, -0.1558837890625, 2.595703125, 0.408447265625, 0.73193359375, -1.4697265625, 2.119140625, -1.119140625, -2.00390625, -0.4560546875, -0.0579833984375, 0.6357421875, -0.412353515625, -0.71435546875, 0.315673828125, -0.44140625, -0.76171875, -1.0419921875, -0.47119140625, -0.4052734375, -0.11041259765625, 0.951171875, -0.02728271484375, 1.283203125, 1.4658203125, 0.365234375, 1.0712890625, 2.4296875, 0.1724853515625, -0.35595703125, 0.8671875, 0.132568359375, 0.68212890625, 0.56103515625, -0.201416015625, -0.075439453125, -1.3759765625, -0.3916015625, 1.455078125, 0.47412109375, 1.4892578125, -1.216796875, 1.0048828125, -0.29248046875, -0.61767578125, 0.352783203125, -0.5966796875, -1.0927734375, -0.11474609375, 0.187255859375, 0.966796875, 0.109130859375, 0.75537109375, 1.095703125, -1.251953125, -0.55419921875, 0.12152099609375, 0.01407623291015625, -1.666015625, -0.273681640625, 0.33740234375, -0.1729736328125, -1.533203125, 0.6796875, -1.7919921875, 1.265625, -1.16015625, 0.4658203125, 0.94970703125, -2.291015625, -0.548828125, 1.7060546875, -0.7939453125, -1.4658203125, 0.1903076171875, 0.71435546875, -0.006946563720703125, -0.251953125, 0.20458984375, 0.79833984375, -2.669921875, -0.06793212890625, 1.595703125, 0.36572265625, -1.05859375, -1.3466796875, -0.2490234375, 0.01458740234375, 1.015625, 1.3623046875, 0.42138671875, -1.67578125, 0.67236328125, -0.0501708984375, -0.37255859375, -1.0634765625, -0.759765625, 0.314697265625, -0.1982421875, -0.60888671875, 0.85791015625, -0.580078125, 0.027557373046875, -0.370849609375, 0.70849609375, 0.323486328125, 0.1602783203125, 0.96142578125, -0.137939453125, -0.330322265625, 2.068359375, -0.039337158203125, -0.515625, -1.43359375, 0.759765625, 0.283935546875, -1.068359375, -0.14111328125, 1.5146484375, 1.05859375, -0.7705078125, 0.1717529296875, -2.0546875, -1.7841796875, 0.061309814453125, -0.9658203125, 1.544921875, -1.02734375, -1.125, 0.59716796875, 1.2021484375, -0.66455078125, 1.537109375, -0.8330078125, 0.3095703125, 1.732421875, -0.10491943359375, 2.197265625, 2.349609375, -0.7919921875, 0.06854248046875, 1.1845703125, 0.050994873046875, 0.77001953125, -0.83984375, -1.1630859375, -0.0115814208984375, 1.470703125, -1.330078125, 1.3310546875, -0.97216796875, -0.2120361328125, -0.254150390625, -0.68212890625, -0.221923828125, -0.3662109375, -1.2861328125, -0.0386962890625, 1.5146484375, 1.271484375, -1.138671875, 0.355224609375, 0.85693359375, 1.8134765625, -1.3125, -0.34716796875, 0.6220703125, 0.87841796875, 0.91357421875, 0.42333984375, -0.351318359375, -2.2421875, -0.043853759765625, -1.1015625, 0.398193359375, 0.69140625, -0.35693359375, 2.669921875, 0.419677734375, -0.114990234375, 2.015625, 0.08050537109375, 0.30224609375, -0.6171875, 0.74609375, -0.5029296875, -1.3994140625, -1.2607421875, -1.2841796875, -0.57275390625, 0.08746337890625, -0.658203125, -0.486083984375, -0.38427734375, 0.385986328125, -1.5849609375, 1.0751953125, 1.51953125, -0.57763671875, 1.0087890625, -0.8486328125, 0.45166015625, 1.087890625, -0.39501953125, 0.0188446044921875, -0.364501953125, 0.8134765625, 0.1920166015625, 1.8359375, 2.16015625, -0.556640625, -0.75927734375, 0.6416015625, -0.10968017578125, -0.7197265625, 0.01085662841796875, 1.5361328125, 1.5, -2.03125, 0.03955078125, -0.0438232421875, 0.06365966796875, 0.7666015625, 0.460693359375, 0.376708984375, -1.958984375, -0.517578125, -1.2001953125, 0.98046875, 1.6220703125, 0.513671875, 0.3125, -0.492431640625, 0.060455322265625, 1.0888671875, -0.05615234375, 2.123046875, 0.5390625, 0.98291015625, -1.2451171875, 0.431640625, -0.286865234375, -0.62060546875, -1.623046875, -1.765625, -0.37548828125, -0.163818359375, 0.300537109375, 0.41845703125, 2.24609375, -0.94140625, -0.292236328125, -0.7958984375, 0.2481689453125, -1.2060546875, -1.2275390625, 0.79541015625, -0.338134765625, -0.98681640625, -0.06756591796875, 0.486572265625, -0.2440185546875, -0.88671875, -1.06640625, -0.98876953125, -1.033203125, -0.6953125, 0.55810546875, 0.197509765625, -0.27734375, 1.0439453125, 0.14111328125, -0.261474609375, -1.1826171875, 0.0858154296875, 0.2393798828125, -0.22705078125, 0.347900390625, -0.72509765625, 0.8310546875, 0.669921875, 0.56640625, -1.375, 0.327392578125, -1.3662109375, 0.9248046875, 0.436767578125, 2.31640625, -0.1690673828125, 0.73388671875, -1.5595703125, 0.5615234375, 0.06646728515625, 2.046875, -1.8486328125, 0.058013916015625, -1.3095703125, 1.0126953125, -0.386962890625, 0.2117919921875, 0.34814453125, -0.2431640625, -0.55859375, -0.1788330078125, -0.32666015625, 0.05615234375, 1.30078125, 1.1142578125, -1.728515625, 0.0684814453125, -0.71875, 0.25927734375, -0.08209228515625, 0.1937255859375, 0.6728515625, 0.4326171875, -0.205810546875, -0.91162109375, -2.01953125, -1.3076171875, 0.57080078125, 1.2236328125, -0.72705078125, -1.1474609375, 0.828125, -1.537109375, 0.343505859375, -0.0740966796875, 1.125, 1.3369140625, -0.94189453125, -0.6533203125, -0.61865234375, -0.57177734375, 0.57421875, 0.8271484375, -0.3388671875, -3.173828125, -0.91357421875, 0.38916015625, -0.6396484375, -1.8544921875, 1.5458984375, 0.46337890625, -2.07421875, -0.56689453125, -0.7470703125, 0.2198486328125, 0.458740234375, 0.420654296875, -0.6240234375, 0.43603515625, -0.009246826171875, -0.431884765625, -0.268798828125, 0.138671875, 1.6025390625, 0.748046875, 0.458251953125, -1.767578125, -0.255859375, 0.2464599609375, 1.4189453125, 0.54052734375, -0.587890625, -0.6533203125, -1.775390625, 2.255859375, -1.193359375, 0.77001953125, -2.072265625, 0.87744140625, -0.09979248046875, -0.29345703125, 0.036712646484375, 0.302978515625, 0.82275390625, -0.85693359375, 0.91064453125, -1.3876953125, -1.189453125, 1.3857421875, 1.8125, -0.04705810546875, 1.2099609375, -0.1624755859375, -1.001953125, -0.71337890625, -0.501953125, 1.8388671875, 1.2587890625, -0.76171875, -0.779296875, -0.54296875, 0.90478515625, -1.6826171875, -0.427734375, -0.46240234375, -0.1971435546875, 0.1287841796875, 0.114501953125, -1.740234375, -1.623046875, 1.220703125, 0.59814453125, 2.052734375, 0.3681640625, 0.33203125, 0.75537109375, 0.88720703125, 0.619140625, -0.1558837890625, 0.31298828125, -0.0909423828125, 1.0146484375, 1.236328125, -0.77978515625, 1.728515625, 0.4814453125, -0.767578125, -1.560546875, 0.302490234375, 2.107421875, -0.380859375, 0.03668212890625, 0.419189453125, 0.412109375, -1.4228515625, 0.91796875, 2.14453125, 0.60498046875, -0.2476806640625, 0.80322265625, -1.7900390625, -1.97265625, 0.666015625, 0.65234375, -0.5849609375, -1.2587890625, -1.2421875, 0.32470703125, 0.1439208984375, 1.123046875, -1.5556640625, 1.8642578125, -0.77197265625, -0.07269287109375, -1.0478515625, 1.0439453125, 0.65283203125, 0.71826171875, -0.7197265625, -0.44287109375, 1.375, 0.326904296875, 2.24609375, -1.12109375, 0.845703125, 0.60009765625, -0.050872802734375, -1.6806640625, 0.61767578125, 1.6357421875, -0.049224853515625, -0.876953125, 1.3251953125, -1.2578125, 0.302490234375, -0.1539306640625, -0.81005859375, 1.158203125, 0.3896484375, 1.8251953125, 0.07769775390625, -1.5322265625, -2.09375, 1.02734375, 1.2275390625, -0.10546875, 0.44970703125, 0.80712890625, -0.888671875, 0.61474609375, 1.828125, 0.374755859375, 0.6943359375, 0.61572265625, 0.03265380859375, 1.794921875, 0.29541015625, 0.568359375, 0.037139892578125, -0.317138671875, -1.3974609375, 1.587890625, 0.3525390625, 1.044921875, 0.29052734375, -1.56640625, -0.01453399658203125, 1.9833984375, 0.284912109375, -0.61328125, -0.1781005859375, -1.0693359375, -1.2861328125, -3.107421875, -0.56689453125, 1.720703125, 0.142578125, 0.7587890625, 1.66796875, 1.9443359375, 0.79052734375, 0.2198486328125, 0.3681640625, -0.2376708984375, -0.33642578125, -1.005859375, 1.78125, -1.0087890625, 0.4033203125, -0.57177734375, -1.4853515625, 0.9765625, -0.3525390625, 0.63037109375, -1.4853515625, -0.40673828125, -0.40673828125, 0.1331787109375, -1.08984375, 0.89111328125, 0.0926513671875, 0.403076171875, -0.767578125, -1.5595703125, -0.185302734375, -0.41650390625, -1.833984375, -0.5166015625, 1.0556640625, 0.50439453125, -0.346923828125, -0.053131103515625, -0.1502685546875, -0.72119140625, -0.89697265625, -1.7763671875, -0.376220703125, 0.7607421875, 0.84619140625, 1.5185546875, 0.79443359375, 2.23828125, 0.97900390625, 0.426025390625, 0.389404296875, 1.3447265625, -0.7060546875, 0.124755859375, -0.6875, 0.280517578125, -0.837890625, -0.607421875, 0.76953125, 0.787109375, 0.06915283203125, 1.7001953125, -0.8173828125, 0.1004638671875, -0.25537109375, -0.85595703125, 1.3330078125, 0.092041015625, -0.685546875, -0.85888671875, -1.8076171875, -1.052734375, 0.464599609375, -0.7041015625, 0.73095703125, -0.81982421875, 0.110595703125, 0.382568359375, -0.6875, 0.07598876953125, -0.71923828125, 0.2103271484375, 1.208984375, 0.66162109375, 0.77734375, 0.6572265625, -2.197265625, -0.64453125, -1.7265625, 0.83642578125, 0.00634002685546875, -0.18994140625, -0.9404296875, 0.1446533203125, -2.302734375, 0.0875244140625, -0.1953125, -2.431640625, -1.9638671875, -1.412109375, -1.9501953125, -0.73583984375, -1.291015625, 0.5615234375, -1.373046875, 0.8291015625, -1.685546875, 0.044097900390625, 0.31689453125, -0.2578125, -0.5205078125, 0.90380859375, 0.455078125, -1.09375, -0.1798095703125, -2.310546875, 0.05963134765625, 1.06640625, 0.0841064453125, -0.4775390625, 0.30224609375, -0.269287109375, 2.17578125, -1.7783203125, -1.51171875, -0.15869140625, -0.0877685546875, -0.361083984375, -0.48095703125, -1.251953125, 0.442138671875, 1.2822265625, -1.900390625, 0.9794921875, 0.54052734375, -0.8154296875, -0.908203125, -0.82177734375, -0.2318115234375, -0.52978515625, -1.8828125, -0.2158203125, -0.1536865234375, -1.32421875, 0.5869140625, 1.48046875, 1.1767578125, 0.1474609375, -0.552734375, -0.59423828125, -0.8447265625, -1.1669921875, 0.1309814453125, -2.1171875, 0.57373046875, -0.425048828125, -1.16015625, -1.31640625, -0.65478515625, 0.89990234375, -0.884765625, -0.60302734375, -0.2939453125, -1.7470703125, 0.1678466796875, -0.751953125, -0.1109619140625, 0.2059326171875, 0.1160888671875, 0.0148162841796875, -0.313232421875, 0.7265625, -0.465087890625, -0.97314453125, -0.1392822265625, 0.54052734375, 1.099609375, 1.0625, 0.90380859375, -0.2020263671875, 0.939453125, 0.35107421875, -0.0254974365234375, 0.63037109375, -0.61376953125, -1.8984375, 0.2098388671875, 0.54052734375, 0.0185089111328125, -1.0908203125, 0.89208984375, 2.55859375, 0.443603515625, 0.1824951171875, 1.185546875, 0.268798828125, 0.609375, 0.12054443359375, -0.51513671875, 0.607421875, 2.412109375, 1.826171875, 0.0252227783203125, 0.69091796875, -0.061370849609375, -0.515625, 1.689453125, 0.184814453125, 1.7734375, 0.8671875, 0.1278076171875, 0.1622314453125, 1.39453125, -0.6650390625, 0.3662109375, -1.2578125, -0.6474609375, 0.07952880859375, 0.7119140625, 1.1845703125, 0.314453125, 0.492919921875, 1.927734375, 0.246826171875, -1.0576171875, -0.44921875, 0.7060546875, -0.69873046875, 1.388671875, 0.0762939453125, 0.10662841796875, 2.244140625, 0.1578369140625, -0.55908203125, 0.413818359375, 0.69775390625, -0.7421875, -0.2822265625, -0.8134765625, -0.5400390625, 0.474609375, -0.475341796875, 1.87109375, -1.423828125, -0.08721923828125, 1.177734375, 0.86669921875, -1.572265625, 0.45361328125, -0.257568359375, 1.25390625, 0.462646484375, 0.45751953125, -0.576171875, -3.03125, -0.54052734375, 0.491455078125, 1.1357421875, -1.5576171875, 0.708984375, 0.11260986328125, -0.87109375, -1.3330078125, 0.0762939453125, -1.498046875, -1.0751953125, 1.408203125, -0.5107421875, 0.1593017578125, 0.4033203125, -2.17578125, -1.18359375, -1.6318359375, 0.52294921875, -0.287353515625, 2.193359375, 1.1640625, 0.456298828125, 0.5283203125, -0.8056640625, -0.411865234375, -0.9013671875, 0.3359375, -1.990234375, 1.8447265625, 1.3232421875, 0.9599609375, 0.227294921875, -1.4794921875, -0.32275390625, 1.8251953125, 0.217041015625, -0.552734375, -0.85302734375, 0.8349609375, -1.9970703125, 1.140625, -1.1708984375, 0.54248046875, -1.9677734375, -0.5, 0.5302734375, -0.2484130859375, 0.822265625, -0.6689453125, -0.354248046875, 0.60302734375, -0.11907958984375, 1.1064453125, -1.92578125, 1.091796875, 0.5927734375, 2.390625, -0.047454833984375, 0.044708251953125, -0.52197265625, -1.4658203125, -1.8837890625, -0.39599609375, -1.9912109375, 1.765625, -0.445556640625, -0.29931640625, -0.01052093505859375, -0.63818359375, -0.86767578125, -0.369873046875, 0.62646484375, -0.4765625, 0.383056640625, -0.95458984375, 0.0870361328125, 0.7880859375, 0.155517578125, 0.04376220703125, -2.203125, -0.1177978515625, 1.197265625, 0.247802734375, -0.9033203125, -0.779296875, 1.16015625, -1.515625, -1.591796875, 0.55810546875, 0.67822265625, 0.56005859375, 1.0341796875, 0.8525390625, 0.69384765625, 0.42919921875, -0.29443359375, -0.0767822265625, -0.931640625, 1.3330078125, 0.54541015625, 0.8505859375, -1.390625, -1.0439453125, 2.1875, 2.740234375, 0.222900390625, 0.421630859375, -1.1435546875, -0.366943359375, 0.277099609375, 0.375, -0.2919921875, 0.6455078125, -0.5986328125, 0.466552734375, 0.004695892333984375, -0.0703125, -1.705078125, 1.029296875, 0.171142578125, -0.58349609375, 0.95263671875, -1.138671875, 0.58203125, -2.35546875, 0.191162109375, -0.2210693359375, -0.059967041015625, -2.0390625, -1.89453125, 0.59033203125, -1.98828125, -0.52685546875, -0.20458984375, 0.135009765625, 0.461669921875, 1.111328125, 0.389892578125, -1.2958984375, -0.335693359375, -1.541015625, 0.62548828125, -1.248046875, 0.365478515625, -1.40625, -0.2393798828125, -1.1103515625, -0.6513671875, -1.2568359375, -0.343505859375, 0.1319580078125, -1.8515625, 2.052734375, 1.380859375, 0.935546875, -0.05841064453125, 1.3349609375, 0.775390625, -1.115234375, -2.65234375, 2.255859375, 1.0, 0.2320556640625, -1.234375, 0.95458984375, -0.318359375, 0.0075531005859375, 1.529296875, -0.330322265625, 0.50244140625, 0.08087158203125, -0.202392578125, 1.0322265625, -0.62939453125, -1.2734375, -1.640625, -3.078125, 1.5009765625, 0.1627197265625, 0.9619140625, 0.313720703125, -0.6396484375, -1.4384765625, 1.8515625, 0.8515625, -0.2115478515625, 0.646484375, -1.1083984375, 3.1484375, -0.12310791015625, 1.3251953125, -2.109375, 0.459228515625, 1.091796875, -0.490966796875, 0.61279296875, 0.1612548828125, 0.2459716796875, -1.7587890625, -1.712890625, -1.0859375, -0.197265625, 0.2288818359375, -0.4501953125, -0.7939453125, 1.34765625, -0.841796875, 1.2119140625, 1.8046875, 1.0673828125, -0.8232421875, 0.2408447265625, -0.144775390625, -0.52978515625, -0.354248046875, -1.291015625, -0.6142578125, 1.291015625, 1.275390625, 0.498779296875, -0.63671875, -0.08111572265625, -0.1690673828125, -0.061309814453125, -0.384765625, -2.14453125, 0.432861328125, 1.068359375, -0.5849609375, -0.65234375, 1.17578125, 1.373046875, 0.67236328125, -0.404052734375, -2.16015625, -0.54736328125, 2.458984375, 0.72705078125, 1.275390625, 0.6708984375, 1.0419921875, -0.276611328125, 0.64599609375, -0.685546875, -0.2374267578125, -1.6962890625, 0.045867919921875, 0.274658203125, -1.5634765625, -0.09844970703125, -0.17236328125, 1.029296875, -1.10546875, -0.11956787109375, -0.406982421875, -0.79248046875, 0.60693359375, 1.9794921875, 2.0390625, 1.306640625, -0.464599609375, -2.4609375, -0.305419921875, 0.6796875, 1.1630859375, -0.76708984375, -0.2064208984375, -0.302734375, 0.82421875, -1.0380859375, 0.272216796875, 0.2293701171875, 0.1492919921875, -0.495849609375, 0.3740234375, 0.44091796875, -0.1641845703125, -0.54345703125, 0.7939453125, -1.955078125, 0.419921875, -0.8447265625, 0.720703125, -0.15283203125, -0.98828125, -0.25244140625, -0.68798828125, 0.3115234375, -0.093505859375, 1.3798828125, 1.3515625, 1.6142578125, 0.394775390625, 1.224609375, 0.6337890625, -0.45458984375, 0.5283203125, 0.82666015625, -1.3798828125, 0.7236328125, 0.354736328125, -0.58203125, 0.7314453125, -0.38232421875, 1.2734375, -1.03515625, -0.98095703125, 0.75146484375, -1.0986328125, -0.279296875, -2.58203125, 0.3935546875, 0.035369873046875, -0.63427734375, 1.0810546875, 0.1334228515625, -0.044189453125, -0.264892578125, 0.62744140625, 0.59765625, 0.62841796875, -0.0289306640625, -0.483642578125, 0.1680908203125, 0.97705078125, 0.244384765625, 0.69677734375, -0.90283203125, 0.01526641845703125, -1.501953125, -0.056915283203125, -0.666015625, -1.0185546875, -0.97802734375, 0.39697265625, -0.37353515625, -1.2353515625, -0.282958984375, -0.0482177734375, 1.548828125, -0.6572265625, -0.004253387451171875, 1.88671875, 0.505859375, -0.54150390625, 0.389404296875, -0.322509765625, -0.96484375, 1.3349609375, 0.478759765625, -1.564453125, 1.060546875, -0.35302734375, -0.99951171875, -0.220947265625, -1.134765625, 0.07861328125, -0.1290283203125, 0.89892578125, 0.3779296875, 0.794921875, 0.85400390625, -0.8623046875, 0.0596923828125, 0.6884765625, -1.119140625, 0.9345703125, 1.142578125, 0.85498046875, -0.11572265625, 0.34423828125, 0.13427734375, 1.861328125, 0.47119140625, -0.402099609375, -0.047332763671875, 0.5390625, -0.736328125, -0.98486328125, 0.1429443359375, -0.46630859375, -0.56982421875, 0.2108154296875, 0.98046875, -0.58447265625, -0.4716796875, -0.28759765625, 0.8076171875, -1.17578125, 1.5361328125, -0.8251953125, -0.93701171875, -0.08001708984375, 2.228515625, 0.50341796875, -0.6953125, -0.59375, -0.2177734375, -0.6044921875, -0.8994140625, 1.322265625, 1.0361328125, -0.4140625, 1.0859375, 2.216796875, -0.344970703125, -0.67333984375, -0.8369140625, 0.5322265625, 0.306640625, -0.5927734375, 0.72021484375, -0.282470703125, 1.3095703125, 0.360595703125, 0.83154296875, 0.0318603515625, -0.434326171875, 0.50634765625, -0.208984375, -0.5537109375, -0.062469482421875, 0.390869140625, -0.9404296875, 0.91064453125, 0.25439453125, -2.75390625, 0.161376953125, 0.451904296875, 0.1910400390625, -0.61083984375, 0.387939453125, 1.1142578125, -0.15869140625, 1.2939453125, -1.8232421875, -0.483154296875, 1.431640625, 1.74609375, -0.57275390625, 0.8134765625, -0.3896484375, 0.5732421875, -0.10150146484375, -2.03515625, 0.66748046875, 0.185302734375, 0.55859375, -0.397216796875, -0.599609375, -0.17578125, -0.87548828125, -0.2978515625, 0.70556640625, 0.7060546875, -0.11669921875, 0.23095703125, -0.139892578125, -1.1787109375, -1.2421875, -0.1956787109375, -0.227294921875, -0.734375, -0.86328125, 0.5576171875, -2.451171875, 0.451171875, 0.0149688720703125, -0.705078125, -0.46875, 1.09375, -0.61669921875, 0.1300048828125, 0.293701171875, -0.763671875, -1.712890625, -1.2001953125, 0.07659912109375, 1.7607421875, -0.1510009765625, -0.1470947265625, 0.1199951171875, -0.7548828125, 0.34521484375, -0.1871337890625, 0.6240234375, -0.2071533203125, -0.365234375, 0.75634765625, 2.4375, -1.0146484375, 1.1328125, 0.416015625, -1.482421875, -0.60546875, -0.56640625, -0.2125244140625, -0.3671875, -0.316650390625, -0.005279541015625, -0.53857421875, 1.791015625, 0.91796875, -0.0012502670288085938, 0.2437744140625, -0.9111328125, 1.2158203125, 2.392578125, 0.97705078125, 0.0655517578125, 1.2841796875, 0.0294647216796875, -0.0771484375, 0.7626953125, 1.2763671875, -0.221435546875, 0.974609375, -0.67431640625, 0.11187744140625, -3.1015625, 1.2119140625, -0.386962890625, -0.216796875, -1.23046875, 0.82568359375, -1.310546875, 0.08612060546875, 0.47607421875, 0.280029296875, -2.529296875, 0.7333984375, 0.50390625, -1.373046875, 0.978515625, 0.27001953125, -0.271240234375, 2.21484375, -0.77880859375, 1.12109375, -0.351806640625, 1.1279296875, -0.06915283203125, -1.5302734375, 0.63916015625, -2.423828125, 1.076171875, -1.0048828125, 1.583984375, -0.568359375, 0.05426025390625, 0.41796875, -1.3154296875, -0.46142578125, -0.3017578125, 0.6279296875, -0.340576171875, 0.342529296875, 1.369140625, 2.7109375, -0.9833984375, -0.509765625, -0.94189453125, 0.87060546875, -3.9453125, -0.0021610260009765625, -0.466064453125, 0.6337890625, -0.053131103515625, -0.60009765625, 2.119140625, 0.7353515625, -0.8076171875, -0.59423828125, -0.37890625, 0.94189453125, -0.067626953125, 0.7529296875, 0.6875, 0.4833984375, -0.07464599609375, -0.67431640625, -0.28466796875, -0.8984375, -1.1083984375, 1.0458984375, 0.04351806640625, 0.658203125, -0.388671875, -0.75439453125, -0.200927734375, 0.1922607421875, 0.6767578125, -2.240234375, 1.224609375, -0.759765625, 0.66357421875, 0.379150390625, -0.7294921875, 1.78515625, -0.470947265625, 0.1446533203125, 1.7119140625, -0.6181640625, -1.2529296875, 1.056640625, 0.51806640625, -0.3798828125, -1.072265625, -0.8056640625, 1.494140625, 1.9091796875, 1.3896484375, 1.1572265625, 0.36328125, -0.00589752197265625, 1.4404296875, 0.312744140625, 0.98876953125, 0.60546875, -1.57421875, 0.67578125, 0.50048828125, 0.6787109375, -0.139892578125, 0.2425537109375, -0.03570556640625, -1.0703125, 0.83544921875, 0.0806884765625, -0.446533203125, -1.92578125, -0.0440673828125, 0.6875, 1.7119140625, 0.11578369140625, -1.5263671875, 0.56494140625, -0.435302734375, 1.083984375, 0.455322265625, 0.034820556640625, -0.22119140625, 0.0401611328125, -1.6826171875, 0.0980224609375, 0.66796875, -0.0190277099609375, 0.783203125, -0.2088623046875, -0.626953125, -1.513671875, 0.186767578125, 0.69189453125, -0.44873046875, 0.974609375, 0.0838623046875, 0.29248046875, 1.15625, -0.460205078125, 0.94873046875, 0.05322265625, 0.0321044921875, 1.0478515625, 1.1982421875, -0.97509765625, -0.7158203125, 0.1729736328125, -2.376953125, 0.12127685546875, -0.6142578125, 1.078125, 0.8642578125, -1.1259765625, -0.5537109375, 0.74462890625, 0.2998046875, 0.8583984375, 1.595703125, -1.1103515625, -0.06390380859375, -1.6640625, 1.345703125, 1.275390625, -0.11334228515625, -0.276123046875, 0.6640625, -1.162109375, -0.177978515625, 0.17626953125, -0.233154296875, 1.4951171875, 0.908203125, 0.344970703125, 0.69921875, -0.197021484375, -0.14453125, -0.91552734375, 0.64306640625, 1.3505859375, -0.91650390625, -0.2305908203125, -1.306640625, 0.51904296875, -2.98828125, 0.6787109375, -0.05047607421875, 0.83935546875, 1.2958984375, -1.8271484375, 0.66064453125, -2.365234375, -1.1142578125, -0.445068359375, -1.6484375, -1.0986328125, -1.375, -2.072265625, -0.01538848876953125, -0.646484375, -0.7373046875, -0.431396484375, -3.009765625, -1.5556640625, 1.80859375, -0.6044921875, 0.9677734375, 1.9521484375, -0.6279296875, 1.8447265625, -2.271484375, -2.32421875, 1.4833984375, -0.537109375, 0.8193359375, 0.9814453125, -0.117919921875, -0.7900390625, -0.99462890625, -0.72705078125, 1.615234375, -0.74853515625, 0.53662109375, -0.76904296875, 0.513671875, 1.9462890625, 0.7080078125, 0.417724609375, 0.685546875, 0.5146484375, 1.2314453125, 0.51806640625, 1.30078125, -1.4228515625, 0.58251953125, -0.7763671875, 1.8427734375, 0.11102294921875, 0.267578125, 1.87109375, 1.5625, 1.3720703125, 0.734375, 1.041015625, -0.4658203125, -0.2215576171875, -0.48583984375, -1.228515625, -0.9658203125, 0.309326171875, -2.1328125, -0.64404296875, -0.1282958984375, 0.2880859375, -0.953125, 0.28662109375, -0.223388671875, -0.435791015625, 0.69873046875, -0.71142578125, 0.471435546875, -0.173095703125, -1.08203125, 0.3857421875, 1.31640625, -0.94580078125, 1.1982421875, 2.693359375, -1.376953125, -0.94140625, 1.3173828125, 0.162841796875, 0.3955078125, -1.02734375, 1.296875, -0.18603515625, -0.771484375, -0.60693359375, 1.2109375, 0.454833984375, 0.048553466796875, -1.0830078125, 0.467529296875, -0.51708984375, 0.5859375, 0.0263824462890625, 1.171875, -1.216796875, -0.2139892578125, 0.59716796875, 0.1668701171875, -1.5673828125, -0.68212890625, 0.8369140625, -1.705078125, 0.91845703125, 0.366455078125, 0.11553955078125, 0.1263427734375, -0.71044921875, 0.53564453125, 1.111328125, 0.45654296875, 0.49365234375, 0.5927734375, -0.76953125, 0.152099609375, -0.157470703125, 1.7861328125, 0.0087432861328125, -0.12213134765625, -0.0018110275268554688, -0.57568359375, -1.2275390625, -0.49072265625, -0.85107421875, -1.9287109375, 0.46142578125, 1.228515625, 1.4814453125, 0.9609375, 2.822265625, -0.482177734375, -0.81201171875, 0.322509765625, -0.9306640625, 0.9580078125, -0.794921875, 0.5615234375, 1.822265625, 2.654296875, -0.81298828125, 0.13720703125, -0.66162109375, 1.6796875, 1.2109375, -1.0185546875, 0.70849609375, 0.794921875, 1.400390625, 1.15234375, -0.1558837890625, 0.19189453125, -0.18798828125, -1.0126953125, 1.912109375, -1.9384765625, -0.33837890625, -0.759765625, -0.33349609375, 0.436767578125, -0.640625, -1.2021484375, 0.347412109375, -1.037109375, -0.53369140625, 1.26171875, -1.8544921875, -2.62109375, 0.4384765625, 0.2548828125, -0.0997314453125, 0.85546875, 0.270751953125, -0.260498046875, -0.50634765625, 1.044921875, -0.740234375, 0.363037109375, -1.2158203125, -0.93798828125, -0.7548828125, 1.0888671875, 0.20263671875, 0.472900390625, -0.65625, 0.3232421875, 1.5439453125, -0.62353515625, -0.80712890625, 1.017578125, -1.2421875, 0.27978515625, 2.044921875, -0.0309295654296875, -1.9033203125, -0.77783203125, 0.161376953125, -0.09686279296875, -0.68603515625, -0.6650390625, 0.24169921875, 0.53857421875, -0.810546875, 0.84033203125, 0.488037109375, 1.3974609375, -0.53173828125, 0.98486328125, -0.81787109375, -1.4580078125, -0.442138671875, -1.962890625, 1.2333984375, -0.281982421875, 0.45703125, 0.47509765625, 0.47314453125, -0.87255859375, -0.9384765625, -0.09619140625, 1.4287109375, 0.64453125, -0.96533203125, 0.42431640625, 0.95654296875, -0.72998046875, 0.27490234375, -0.12017822265625, -0.2381591796875, -0.00888824462890625, 1.373046875, 0.8076171875, -1.21484375, -0.8896484375, 0.12164306640625, -1.193359375, -0.7431640625, 1.9541015625, 2.13671875, 1.0126953125, -0.1434326171875, -1.7353515625, -0.56689453125, -0.324462890625, 0.9140625, 1.302734375, 0.24951171875, 0.81982421875, -0.92822265625, -0.419189453125, -0.11883544921875, -0.1484375, 1.7890625, -1.8056640625, 0.744140625, 0.05224609375, -0.7607421875, 0.4052734375, -1.068359375, -0.74609375, 0.78857421875, 0.962890625, 1.291015625, 2.13671875, -0.437744140625, -1.7275390625, -1.5439453125, -1.4375, 0.7431640625, 1.37890625, -0.1904296875, 1.0244140625, -1.2529296875, 1.8427734375, 0.4560546875, -0.4453125, -0.446044921875, -0.5888671875, 0.0943603515625, -0.6875, -0.6279296875, -0.55126953125, -1.9501953125, 0.57421875, 0.36279296875, -1.05078125, -1.138671875, 1.525390625, -2.16015625, 1.21875, -0.040496826171875, 0.486572265625, 0.919921875, -0.720703125, 0.468994140625, -1.5009765625, -1.8154296875, -2.072265625, -0.272216796875, -1.37109375, -0.5849609375, 2.171875, -0.9609375, -0.06292724609375, -0.409423828125, -1.0234375, 0.27197265625, 0.51904296875, -1.01171875, 0.5908203125, -0.2406005859375, -1.4658203125, 0.169189453125, 1.041015625, 0.274169921875, 0.64111328125, 1.63671875, 0.033233642578125, -0.54296875, -0.0887451171875, -0.82568359375, 0.73876953125, -1.3349609375, -1.08984375, 1.3935546875, -1.044921875, 0.004119873046875, 0.452392578125, 0.057586669921875, -0.2548828125, -1.2353515625, 0.287109375, -1.3984375, 0.06646728515625, 1.0439453125, 0.492919921875, 1.6650390625, 0.59912109375, -0.203369140625, 0.5830078125, -2.990234375, 0.2491455078125, 0.224609375, 0.7216796875, -1.736328125, 0.73876953125, 2.064453125, 2.009765625, -0.6591796875, -0.4296875, -0.209228515625, -0.380615234375, 0.5, 0.59375, 0.467041015625, 0.6884765625, -0.58935546875, 0.470947265625, 0.748046875, 0.9404296875, -0.50830078125, -0.87451171875, 1.5244140625, 0.36572265625, -0.60888671875, -0.32861328125, -0.1458740234375, 0.71533203125, -0.81005859375, -0.501953125, 0.1207275390625, -0.25146484375, 0.6826171875, 0.58447265625, 0.45849609375, 0.67236328125, -2.287109375, 0.326904296875, 1.9609375, -0.69921875, -1.2919921875, 1.087890625, 1.6533203125, -0.91259765625, -1.1162109375, 1.8974609375, 0.95751953125, 0.064453125, 0.45166015625, 0.6337890625, -2.080078125, -0.043304443359375, -0.2142333984375, 0.58837890625, -0.2425537109375, -0.68408203125, -0.2020263671875, 1.056640625, -0.59033203125, -0.61328125, 0.97216796875, 1.158203125, 0.352294921875, -1.3486328125, -1.40625, 0.638671875, -0.609375, 0.513671875, 0.212890625, 0.5302734375, 2.197265625, 0.9921875, -0.2919921875, 2.7421875, 1.4814453125, 0.1759033203125, 0.4248046875, -1.697265625, 0.392578125, 1.7119140625, -0.301025390625, 1.3388671875, 1.375, -0.67041015625, -0.2008056640625, 1.2080078125, -0.13427734375, -0.238525390625, 0.325439453125, -0.306640625, 0.54443359375, 1.498046875, -0.09332275390625, 0.285400390625, -0.1485595703125, 2.650390625, 0.11602783203125, 0.6181640625, 0.89697265625, 0.77587890625, 0.50390625, -1.109375, 1.5068359375, 0.505859375, -2.130859375, 0.27294921875, -1.5556640625, 0.51611328125, 0.615234375, 0.654296875, 0.86376953125, -0.25927734375, 1.03125, 0.07440185546875, -0.2110595703125, -0.35546875, 0.39697265625, 0.97119140625, -0.187744140625, 1.318359375, -0.58349609375, -1.1142578125, -0.87255859375, 0.308349609375, 0.5966796875, -0.309326171875, -1.751953125, 0.564453125, 0.435546875, -0.57177734375, -0.29345703125, -1.99609375, -0.186279296875, -2.3046875, 0.98193359375, -0.646484375, -0.32177734375, -0.259033203125, -0.1624755859375, -0.5947265625, 0.6728515625, -1.22265625, 0.4814453125, 0.152587890625, -0.5087890625, 0.89208984375, -0.284912109375, 0.90673828125, -0.0999755859375, -1.330078125, 0.90966796875, 0.99365234375, 0.44921875, -0.7158203125, -0.9423828125, -0.755859375, 0.354736328125, 1.7685546875, -0.55712890625, 1.287109375, -1.8291015625, 0.424560546875, 0.6162109375, -0.74365234375, -0.6591796875, 0.48291015625, -0.406005859375, -0.00839996337890625, 1.1630859375, -1.537109375, -1.291015625, -1.201171875, -0.162109375, -1.00390625, -0.26513671875, -0.11688232421875, 1.087890625, -0.3056640625, 1.4091796875, -0.3525390625, 1.158203125, 0.6103515625, -0.7626953125, 0.833984375, 1.3994140625, -0.84521484375, -0.470703125, -0.7373046875, 1.2783203125, -0.10009765625, -0.822265625, 0.44287109375, -1.2568359375, 0.92626953125, 0.1776123046875, 0.09033203125, -0.55712890625, 0.19482421875, -0.496826171875, -1.431640625, 0.724609375, 0.7060546875, -1.115234375, 1.408203125, 0.30029296875, -0.2060546875, -0.97802734375, -1.236328125, 0.72119140625, -0.033294677734375, -0.209716796875, -1.2373046875, 0.57568359375, -0.282470703125, -2.55859375, -0.0997314453125, 0.9150390625, 0.85400390625, 1.0791015625, -1.138671875, -0.324951171875, 0.05206298828125, 0.52734375, -0.87060546875, -1.6796875, -0.67333984375, 0.10772705078125, -0.2440185546875, -0.88134765625, -0.8994140625, 0.60205078125, -0.012054443359375, 0.583984375, 1.2783203125, 0.5087890625, -0.65673828125, -0.0531005859375, -0.493896484375, -1.1123046875, -0.32568359375, 0.90771484375, -0.399658203125, -0.4140625, 0.306640625, 1.931640625, -0.2332763671875, 0.138916015625, 1.0205078125, -0.16015625, 1.3359375, -1.5400390625, -0.712890625, -0.0009851455688476562, 0.42626953125, -1.8896484375, 0.268310546875, 0.31201171875, 0.185302734375, -1.576171875, 1.02734375, 1.033203125, -0.5869140625, 1.2587890625, 0.6982421875, 1.08984375, 0.262939453125, 0.1827392578125, -0.6494140625, -0.1849365234375, 0.462646484375, -1.521484375, -0.67138671875, -0.277099609375, 0.08624267578125, -0.12371826171875, -2.857421875, -1.2412109375, 0.3916015625, -2.103515625, -1.1787109375, -0.08447265625, 0.463134765625, -0.7841796875, -1.171875, -1.3720703125, -0.1363525390625, -0.2490234375, -0.246337890625, -0.0816650390625, -0.1337890625, 0.14892578125, -0.34423828125, -0.47216796875, 0.294677734375, 0.00893402099609375, 1.0654296875, 0.58984375, -0.269287109375, 0.65478515625, -1.5546875, 1.0966796875, -0.00038433074951171875, 1.3935546875, -0.6728515625, 0.45654296875, -0.74169921875, 0.2215576171875, -0.61865234375, 0.45556640625, 0.443603515625, 1.7783203125, 1.7177734375, -2.287109375, -0.818359375, -0.335205078125, -0.986328125, -0.294677734375, -0.422607421875, -2.16796875, 1.1279296875, -0.763671875, -0.252197265625, 0.56689453125, -1.7236328125, -0.8251953125, -0.435791015625, -0.708984375, 0.62158203125, 0.365234375, -0.0966796875, -0.14794921875, -1.7587890625, -0.978515625, 2.361328125, -1.0283203125, 1.8896484375, 1.51953125, 1.06640625, 1.7734375, 1.20703125, -1.68359375, -0.054901123046875, 1.322265625, 0.70849609375, 1.2392578125, 0.99609375, -0.96337890625, 1.3447265625, -0.99267578125, 0.291015625, 0.42626953125, 0.66552734375, 0.18798828125, 1.359375, 2.599609375, 1.2998046875, -0.8427734375, -0.8818359375, 1.9580078125, -1.0048828125, 1.28515625, 0.734375, 1.9296875, -0.369873046875, -0.062744140625, 0.364013671875, 0.480224609375, 1.3310546875, -1.150390625, 0.302734375, 0.6142578125, -0.1478271484375, 0.0416259765625, 1.4638671875, -1.8515625, -0.18505859375, -0.7607421875, 0.69384765625, -0.11907958984375, -0.428955078125, 1.240234375, 0.2139892578125, -0.72998046875, 0.548828125, -0.0789794921875, 0.357177734375, -1.0087890625, 0.12322998046875, -0.1552734375, -0.23779296875, 1.5, -0.95947265625, -0.1109619140625, 1.443359375, 1.365234375, -0.5732421875, -0.640625, -0.42333984375, -0.339599609375, -0.226318359375, -0.58740234375, 1.68359375, 0.82666015625, -0.72021484375, 0.489990234375, -2.119140625, 0.01934814453125, 2.00390625, -0.513671875, -2.55078125, -0.0445556640625, 1.4091796875, 1.5263671875, 0.67431640625, 1.453125, -0.64892578125, -1.2578125, 1.072265625, 0.67529296875, 0.91650390625, -0.11322021484375, -1.2744140625, 0.005397796630859375, -0.5078125, -0.740234375, -0.62939453125, -0.4375, 0.36865234375, 1.1376953125, 0.28271484375, -0.1068115234375, 0.3916015625, 0.41455078125, 0.453125, 1.30859375, -0.0379638671875, 0.73876953125, 0.45166015625, 1.859375, 0.1729736328125, -0.4365234375, -0.394287109375, -1.2265625, 0.6123046875, 0.51513671875, 1.12890625, 0.7763671875, -0.16845703125, 0.044036865234375, 0.481201171875, -2.314453125, 0.0833740234375, 1.4267578125, 0.41064453125, -0.57421875, -0.310791015625, -2.296875, -0.77587890625, -1.8427734375, 1.162109375, 0.79833984375, -0.2056884765625, 0.1705322265625, -1.599609375, -1.419921875, 1.6806640625, 0.28125, 0.1055908203125, -0.8447265625, 0.3134765625, -0.1885986328125, -0.1724853515625, 1.0390625, -0.125244140625, 0.042999267578125, -0.0928955078125, -0.32470703125, 0.9775390625, 0.167236328125, -0.72998046875, 0.472412109375, -0.90185546875, 2.1171875, 1.19921875, 0.38525390625, -0.705078125, -1.44921875, 0.445556640625, -1.0966796875, -0.12005615234375, -0.888671875, 0.427001953125, 0.96923828125, 0.352294921875, 0.364013671875, -1.375, -0.2744140625, -0.1868896484375, 0.390869140625, 0.7578125, 0.311279296875, -0.395751953125, -0.466796875, 0.5732421875, -0.35595703125, -1.994140625, 0.1688232421875, 0.328857421875, 0.08367919921875, 2.3203125, 0.94580078125, -0.10943603515625, -0.454833984375, -0.431396484375, 0.74853515625, -1.2783203125, -1.837890625, -0.0604248046875, -0.0011377334594726562, -0.0885009765625, -0.7021484375, -0.71484375, -1.7265625, 0.77099609375, 0.1861572265625, -0.62255859375, -0.362060546875, 0.22802734375, 0.7900390625, 0.2763671875, -0.599609375, -1.1279296875, 1.6025390625, -0.2890625, -1.1611328125, 1.359375, 1.26171875, 1.4541015625, 0.9501953125, -0.186279296875, -0.1663818359375, 1.0810546875, -0.47802734375, -1.982421875, 0.92138671875, 0.2071533203125, 1.083984375, -0.84912109375, -0.07049560546875, 0.2198486328125, -0.71630859375, 1.361328125, -0.82373046875, 0.03802490234375, -0.583984375, -0.406494140625, 0.77783203125, -0.205078125, -0.1702880859375, 1.33984375, 1.365234375, -0.10443115234375, 0.454345703125, -3.07421875, -1.26171875, -0.74462890625, -0.0487060546875, -1.5546875, -1.1044921875, -1.0673828125, 2.107421875, -1.138671875, -0.69482421875, 0.1612548828125, 0.27392578125, -0.763671875, -0.81201171875, 0.4228515625, -0.33251953125, -0.94287109375, 0.9228515625, -0.452880859375, 0.1717529296875, -1.009765625, -0.91259765625, -1.2724609375, -1.2861328125, 0.26025390625, 0.64013671875, -1.05078125, 1.361328125, -0.3046875, 0.951171875, 0.75244140625, -0.0643310546875, -0.89892578125, 0.87744140625, -0.1845703125, -1.517578125, 0.8798828125, -0.88720703125, -0.5380859375, 0.66748046875, -0.1435546875, 0.3134765625, -0.48876953125, 0.720703125, -1.8720703125, 1.4052734375, 0.05950927734375, -0.488525390625, 1.0712890625, -0.34619140625, -1.162109375, 0.07281494140625, 0.60205078125, 0.55859375, 0.261962890625, -0.17919921875, -2.291015625, 0.439208984375, 0.69091796875, 0.646484375, 1.2109375, -0.06439208984375, 0.2861328125, -1.068359375, -0.260498046875, 1.59765625, 0.1378173828125, -0.10797119140625, 2.16015625, -1.875, -0.42919921875, -0.189208984375, -0.52978515625, -0.95654296875, 0.2435302734375, -0.86376953125, 0.69873046875, -0.490234375, -0.1024169921875, -0.97998046875, -1.24609375, 1.4462890625, 2.271484375, 0.0699462890625, 0.5234375, -1.431640625, 0.0175628662109375, 0.186279296875, -0.41650390625, -0.315185546875, 2.119140625, -1.9833984375, -0.57421875, 0.556640625, 0.11358642578125, -1.208984375, 0.2064208984375, 0.10772705078125, 0.6591796875, 0.68359375, 1.34375, 0.93310546875, 1.626953125, 0.53271484375, 1.4794921875, -0.1466064453125, -0.237060546875, -0.65234375, -0.85986328125, -0.10113525390625, -0.703125, -0.94580078125, -1.3564453125, -0.28515625, -0.5146484375, -0.254638671875, 0.88720703125, -0.69921875, -1.6025390625, 0.2325439453125, 0.7421875, 1.12109375, 0.08905029296875, 0.4814453125, 1.72265625, -0.402099609375, -1.6513671875, 1.150390625, -0.44921875, 1.771484375, 0.0809326171875, 1.8994140625, -0.024383544921875, 0.01502227783203125, -0.6435546875, 0.65185546875, -0.70654296875, 0.923828125, 0.82373046875, -0.86474609375, -1.5537109375, -0.62451171875, -0.0679931640625, -0.21728515625, 0.38671875, 2.607421875, -0.875, 1.443359375, -0.3681640625, 0.51953125, -1.8759765625, 0.22802734375, -0.84326171875, 0.89501953125, -1.6181640625, -0.127685546875, 0.04595947265625, -0.88427734375, -1.09765625, 0.317138671875, 0.91552734375, 0.6416015625, -1.3232421875, 1.3310546875, -1.396484375, -0.775390625, -0.405517578125, 0.288818359375, -0.383544921875, -0.71044921875, 1.0859375, 1.1142578125, 0.17822265625, 1.46875, -1.138671875, 0.2841796875, 3.01171875, 0.370849609375, 0.55810546875, 1.0546875, -0.461181640625, 0.26416015625, 0.103759765625, 0.08685302734375, 0.87890625, 0.06591796875, 0.830078125, 1.21875, 0.0020389556884765625, 1.5947265625, 0.479736328125, 0.77392578125, -1.8154296875, 0.54833984375, 1.451171875, 0.40283203125, -0.391845703125, 0.0182647705078125, -1.27734375, -0.266845703125, -2.533203125, -0.43359375, 0.292236328125, -0.77978515625, -0.28173828125, 1.5810546875, -1.1337890625, 0.175537109375, 1.31640625, 0.66455078125, -0.9365234375, -2.326171875, -0.1572265625, -0.1905517578125, 0.0413818359375, 0.8369140625, -1.802734375, -0.56591796875, 1.16796875, -0.311279296875, -0.9326171875, -0.51416015625, 0.437255859375, 0.408447265625, -0.51708984375, -0.84033203125, 1.220703125, 1.0869140625, 0.4130859375, 1.6826171875, 0.55419921875, -1.2021484375, 0.625, -0.215087890625, 1.2353515625, 0.32373046875, -0.685546875, 0.935546875, 1.1416015625, -0.4208984375, 0.33251953125, -0.53076171875, -0.357177734375, 0.24853515625, 0.398193359375, -0.5576171875, 0.5185546875, -0.6767578125, -1.6875, 1.419921875, -0.4052734375, -2.646484375, 1.06640625, 0.84521484375, -0.74609375, 0.560546875, 0.69970703125, 0.546875, -0.63623046875, 0.291015625, 1.8515625, -0.266845703125, 1.291015625, -0.0307464599609375, -0.54296875, 0.8818359375, 0.666015625, 0.3134765625, -1.53125, -1.1005859375, 0.95751953125, -0.7041015625, 0.64306640625, 0.611328125, 0.0802001953125, -0.765625, -0.1015625, 1.5146484375, 1.0888671875, 1.4677734375, 1.0673828125, -2.46875, -0.247314453125, -0.0012722015380859375, 0.1549072265625, 0.078125, 0.07183837890625, -0.439453125, 0.0687255859375, -1.1025390625, -0.3818359375, -0.625, 0.2110595703125, -0.41943359375, -0.2432861328125, 1.0615234375, -0.6552734375, 0.425048828125, 0.482421875, 0.304443359375, -1.0556640625, -0.139404296875, -1.025390625, -0.0238189697265625, 0.19921875, 0.9111328125, -1.119140625, 1.0283203125, -1.087890625, 0.6064453125, 0.623046875, -0.372802734375, 0.2216796875, 0.65478515625, 0.5537109375, -1.0146484375, -0.4658203125, 0.2171630859375, -1.4541015625, -1.423828125, 0.289794921875, -0.94091796875, 0.224609375, -0.07611083984375, -0.137451171875, -0.69091796875, 0.91552734375, -0.191650390625, -0.86865234375, -0.2061767578125, 1.732421875, -0.609375, -0.603515625, -0.56982421875, -0.53369140625, -0.059173583984375, -0.673828125, 0.0966796875, 0.406982421875, 0.75830078125, 0.3798828125, 0.1368408203125, -0.36376953125, -1.0576171875, -1.7705078125, -0.69287109375, -0.192138671875, 0.93701171875, -1.189453125, 0.64892578125, -0.353515625, -0.99267578125, -0.98046875, 1.6279296875, -0.294677734375, 0.6767578125, -1.2392578125, 1.28125, -0.56640625, 0.75341796875, 2.220703125, 2.431640625, 0.626953125, 0.496337890625, 0.44091796875, 2.11328125, -0.7685546875, -0.425537109375, 1.2841796875, 0.85986328125, 0.5537109375, 2.71875, -0.2252197265625, 1.166015625, 0.0911865234375, 0.14697265625, 2.236328125, 0.74658203125, -0.947265625, -0.2010498046875, 0.44921875, 0.5908203125, 0.454345703125, 0.43701171875, 0.91796875, 0.52880859375, -0.7001953125, -0.5478515625, 1.341796875, -1.841796875, -1.1796875, -0.29052734375, 0.52685546875, -1.7080078125, 0.171142578125, 0.164794921875, -0.52099609375, 0.24072265625, -1.005859375, -0.95556640625, -1.2626953125, -0.011688232421875, -1.1025390625, -0.35107421875, 0.0882568359375, 0.057525634765625, -1.619140625, -1.8154296875, 0.52734375, 0.716796875, 0.9765625, -0.2568359375, 1.107421875, 0.87548828125, -0.066650390625, -1.490234375, -0.43017578125, -1.9765625, -0.525390625, 1.2978515625, -1.15625, 1.2822265625, 0.83935546875, 0.38134765625, -0.257080078125, -0.5439453125, -1.58984375, -1.2587890625, -1.193359375, -0.09393310546875, 0.3046875, -0.08233642578125, 2.0, 1.5693359375, 0.6123046875, -0.54150390625, 0.216796875, -0.1177978515625, -0.77880859375, 0.55078125, -0.309814453125, -0.947265625, -1.3720703125, 0.68896484375, 0.243408203125, -0.91748046875, -0.4384765625, 0.1434326171875, -1.884765625, 0.529296875, -0.2369384765625, -0.41650390625, 1.5654296875, 0.5947265625, 1.658203125, 0.892578125, 0.0498046875, -1.556640625, -0.60546875, -0.2445068359375, -0.0017328262329101562, 0.138671875, -0.4384765625, -1.27734375, -0.327880859375, -0.312255859375, 2.619140625, -0.69091796875, -0.66748046875, -0.59619140625, -0.344482421875, -0.09661865234375, -0.265380859375, -1.06640625, 1.1279296875, 0.2169189453125, -0.49560546875, -0.66845703125, 0.64306640625, 0.72314453125, 2.189453125, -1.1376953125, 0.96240234375, -0.2130126953125, -0.447509765625, -1.05859375, -0.1260986328125, -0.401123046875, 0.8525390625, 1.146484375, -0.63525390625, 0.5830078125, -0.1976318359375, -0.027099609375, 0.99755859375, -1.6337890625, -1.8916015625, -0.461181640625, 3.705078125, 0.377685546875, -0.197021484375, -0.3935546875, 1.05078125, -0.54736328125, 0.6708984375, 0.2061767578125, 1.65234375, -0.030181884765625, -0.2371826171875, -0.73828125, 1.306640625, -0.08087158203125, 1.19921875, -1.90625, 1.5205078125, -0.239501953125, -0.07928466796875, -0.7080078125, 0.67724609375, 0.92724609375, 0.0159454345703125, -0.71728515625, 1.234375, -0.297607421875, 1.599609375, -0.55615234375, -1.52734375, 1.9912109375, 1.5068359375, 1.9248046875, 0.74609375, 0.91357421875, 0.61181640625, -0.481201171875, 0.8974609375, -1.1298828125, 1.0205078125, 1.4716796875, 0.050506591796875, 0.27978515625, 1.7568359375, 0.732421875, -0.051666259765625, 0.88427734375, 1.744140625, -0.23095703125, -1.9384765625, -0.6435546875, -0.58251953125, 0.619140625, 0.52197265625, 1.0439453125, -0.5341796875, -0.0095367431640625, -1.2275390625, -1.27734375, 0.1456298828125, 1.666015625, 1.484375, -0.0173492431640625, -0.720703125, 1.1572265625, -1.6328125, 0.67822265625, -0.51708984375, -1.482421875, -1.2529296875, -0.51171875, -0.64892578125, -0.923828125, -1.7294921875, -0.74267578125, 0.0355224609375, 1.7431640625, -0.265869140625, 1.00390625, -0.3583984375, -0.24951171875, -0.84521484375, 2.130859375, 0.266357421875, 1.806640625, -0.72119140625, 0.41845703125, 2.1953125, -0.08221435546875, -0.1646728515625, 0.2900390625, -1.748046875, 0.304931640625, -1.03515625, -0.50341796875, 0.896484375, 1.1611328125, 1.3427734375, -0.2476806640625, 1.0625, -0.1810302734375, -0.281005859375, -0.021514892578125, 1.2021484375, -1.55078125, 1.1533203125, -1.8671875, -0.67578125, 1.0341796875, -0.177001953125, 0.6728515625, -0.52685546875, 1.0625, -1.1474609375, 0.86572265625, -0.9853515625, -0.41845703125, 1.4912109375, 0.428955078125, -0.20556640625, -0.019439697265625, 1.111328125, -2.796875, -0.552734375, 0.2325439453125, 0.9677734375, 1.0322265625, -1.2939453125, -1.341796875, -0.72265625, 0.8310546875, -1.013671875, 0.2474365234375, -0.65234375, 0.66064453125, -0.47607421875, -0.1348876953125, -0.413330078125, 0.1910400390625, -1.3349609375, 0.13623046875, -1.1982421875, 1.064453125, -0.93505859375, -0.65478515625, -0.517578125, 1.513671875, 1.529296875, -0.060211181640625, 0.495361328125, -2.775390625, 1.2998046875, -0.68505859375, 0.02362060546875, 0.2724609375, -2.35546875, -0.139404296875, -0.8330078125, 1.18359375, -0.346435546875, 1.2119140625, -1.4189453125, -1.330078125, 0.4638671875, -1.03125, 1.451171875, -0.178466796875, 0.6416015625, -1.0908203125, -0.61865234375, -0.60791015625, -1.4375, -0.09344482421875, 1.8291015625, -1.181640625, -0.497802734375, -1.0224609375, 0.38037109375, 0.250732421875, -0.1651611328125, -0.46435546875, -0.29345703125, 0.2393798828125, -1.609375, 1.240234375, 0.416015625, -0.8837890625, 0.200439453125, 0.0762939453125, 0.54248046875, -1.6630859375, 0.3818359375, 0.46875, 0.9931640625, 1.1103515625, -1.4794921875, 0.6630859375, -0.6220703125, 0.62109375, 0.370361328125, 0.0262603759765625, -0.33447265625, -1.3642578125, -0.55322265625, 0.404296875, -0.50732421875, -1.7578125, 0.77294921875, 0.00598907470703125, -2.6953125, 1.53515625, 0.215576171875, -0.44873046875, -0.5234375, 0.399169921875, -2.771484375, -0.1434326171875, -2.013671875, -0.287109375, -1.1103515625, -0.10107421875, 0.337646484375, 1.6845703125, 1.9990234375, -1.5537109375, -0.61376953125, 0.12237548828125, 1.3896484375, 0.5810546875, 0.401123046875, -0.344970703125, 1.6865234375, 0.76513671875, 0.7587890625, 0.4150390625, -1.169921875, -0.214599609375, 0.56298828125, 0.7587890625, -0.0205078125, -0.62451171875, 1.1435546875, -1.3037109375, 0.042449951171875, -1.47265625, 0.90185546875, 1.095703125, 1.3134765625, 1.005859375, 1.1787109375, -0.65185546875, -0.181640625, 0.7744140625, 1.5576171875, 1.109375, 0.61181640625, 1.2919921875, 0.86181640625, -0.2198486328125, 0.1612548828125, 1.259765625, 0.60205078125, 1.021484375, 1.615234375, -0.1492919921875, 0.59228515625, -0.5859375, -0.541015625, -0.229248046875, 0.61328125, 1.091796875, -0.42578125, 0.275146484375, -1.47265625, -0.267333984375, -0.262939453125, 1.2001953125, -0.72265625, 0.56640625, 0.1297607421875, -1.66796875, -1.5126953125, -1.90625, 2.95703125, -0.428466796875, -0.09747314453125, -1.068359375, -0.748046875, 0.68798828125, -1.716796875, 0.92041015625, 0.5234375, 0.254638671875, 1.4697265625, 1.3515625, 0.857421875, -0.06512451171875, -1.0927734375, -0.148193359375, -1.005859375, 0.18701171875, -0.5830078125, 0.299072265625, 0.1932373046875, -0.75439453125, 0.0184478759765625, 0.521484375, -0.283203125, -0.289306640625, -0.1806640625, 0.14990234375, -0.955078125, -1.1611328125, -2.087890625, -1.431640625, 0.888671875, -1.4482421875, 0.900390625, -1.1484375, -0.0011053085327148438, -0.397216796875, 1.0146484375, -0.0872802734375, 0.76953125, -0.376953125, 0.615234375, -1.1708984375, 2.1875, -0.86328125, -0.4111328125, 0.36376953125, -0.6767578125, 0.90576171875, -1.234375, -0.237548828125, 0.053009033203125, -0.52587890625, -0.210205078125, -1.0703125, -0.2376708984375, 0.59619140625, -1.4443359375, -0.294677734375, 0.51904296875, -0.359619140625, 1.4873046875, -0.892578125, -1.083984375, 1.0751953125, -0.343017578125, -0.10528564453125, 1.08203125, 0.1815185546875, -0.984375, -0.88427734375, -0.204833984375, -0.2340087890625, 1.1220703125, -0.478271484375, 0.37255859375, 0.92626953125, 1.5595703125, 0.69287109375, -0.0487060546875, -0.2037353515625, -2.18359375, -3.125, 2.1640625, 0.90234375, 0.27783203125, 2.091796875, 0.62646484375, 1.2041015625, 0.150390625, 1.01171875, -0.70849609375, -0.59765625, -0.32373046875, 0.68896484375, -0.73583984375, 0.0992431640625, 0.134033203125, -0.6171875, 0.0086212158203125, 1.7197265625, 0.438720703125, -0.5615234375, 1.8466796875, 0.50927734375, 0.2301025390625, 0.116943359375, -0.4248046875, -0.4228515625, 0.51025390625, 0.099609375, -0.83544921875, -0.2354736328125, 0.2493896484375, 0.47509765625, 0.3955078125, 1.2919921875, -1.5986328125, 1.5380859375, -0.12176513671875, -0.01450347900390625, 1.7021484375, 0.21142578125, -0.76025390625, -0.168212890625, -0.23095703125, -0.7783203125, -0.0296783447265625, 0.58935546875, -0.75341796875, 0.369140625, -0.06317138671875, -0.04180908203125, 0.82275390625, -2.234375, 0.2005615234375, 0.175048828125, 0.042327880859375, -0.6708984375, 1.8056640625, -0.11810302734375, 1.2294921875, 0.88427734375, 1.2275390625, 0.33154296875, -1.859375, 1.212890625, -1.017578125, 1.142578125, -0.71337890625, -0.425537109375, -1.033203125, 1.2919921875, -0.489013671875, -0.050994873046875, -0.2042236328125, -0.62548828125, 0.64697265625, 1.0791015625, -0.80615234375, -0.83447265625, 1.69921875, 1.38671875, 0.65478515625, -0.482666015625, -0.68359375, 0.4560546875, 1.314453125, 0.490478515625, -0.595703125, -3.18359375, -0.82177734375, -0.2122802734375, -0.41845703125, -0.8681640625, 0.92578125, -0.84716796875, 0.365478515625, -1.3037109375, 0.206787109375, 1.576171875, 0.8740234375, 0.50244140625, -2.345703125, 0.77294921875, -0.93798828125, 2.912109375, 0.251708984375, 0.10589599609375, -1.2900390625, -1.595703125, 2.21484375, -0.10546875, 1.216796875, -0.35498046875, 0.7548828125, 0.8828125, -1.7392578125, -0.70458984375, 0.486572265625, 0.83154296875, -0.56201171875, 0.61865234375, -0.1473388671875, 0.200927734375, -0.0435791015625, 0.20849609375, -0.61767578125, 0.370361328125, -0.93408203125, -0.57763671875, -0.285400390625, 0.1287841796875, -1.1015625, -0.2425537109375, 0.043487548828125, 1.7421875, -0.09197998046875, -0.153564453125, -1.759765625, -0.90966796875, 0.37353515625, 0.5830078125, 0.379638671875, 2.080078125, 0.271240234375, 0.90869140625, 1.19921875, -0.6923828125, 0.806640625, 0.354248046875, -0.2088623046875, -0.62060546875, -0.98779296875, 2.62890625, -0.06353759765625, 1.27734375, 0.14794921875, 1.2265625, 1.08203125, 1.3759765625, -0.278076171875, -0.86328125, -0.552734375, -0.92626953125, -2.076171875, 2.328125, -0.1414794921875, 1.1640625, 1.083984375, -0.759765625, 1.3916015625, -2.34375, -0.447265625, -0.4111328125, 0.11224365234375, -1.5712890625, -1.9814453125, 1.72265625, -0.094970703125, -0.09503173828125, 1.3974609375, -0.9306640625, -1.3076171875, -0.212890625, -0.499755859375, -0.30322265625, -2.26953125, 0.81689453125, 0.14990234375, 0.448486328125, -0.255859375, -0.39306640625, 1.2958984375, 0.00879669189453125, 0.15185546875, -1.82421875, 0.77587890625, -0.21728515625, 0.5595703125, -0.0241241455078125, -0.4462890625, -0.3994140625, -1.0009765625, 0.305419921875, 1.8798828125, -2.357421875, 0.61083984375, -2.154296875, -0.8525390625, 1.814453125, 1.279296875, -1.4111328125, 0.47705078125, 1.931640625, 0.207275390625, -0.4287109375, 0.3037109375, -1.2041015625, -1.453125, 1.6318359375, 1.0419921875, 2.416015625, -1.328125, 0.391357421875, 0.1943359375, -1.9560546875, -0.295166015625, 2.119140625, 0.62158203125, -1.0146484375, -0.331298828125, -0.6669921875, -1.39453125, 0.405517578125, -0.9384765625, 0.96142578125, 0.043212890625, -0.451416015625, -0.78271484375, 0.82763671875, -0.292724609375, -0.22265625, 0.82275390625, 0.9130859375, 0.7548828125, 0.81396484375, 0.403564453125, 0.11688232421875, -0.65966796875, -0.0157012939453125, 0.58056640625, -1.10546875, 0.388671875, 1.6328125, -0.61572265625, 0.912109375, 0.352783203125, -0.90625, 0.337646484375, 0.5263671875, -1.21875, -1.796875, -0.6533203125, 0.138916015625, -0.322265625, -0.66357421875, 0.64794921875, 0.477294921875, -0.2320556640625, 0.6435546875, -0.89794921875, 0.0648193359375, -0.08673095703125, -0.315185546875, -1.802734375, 0.3955078125, 1.2802734375, -0.619140625, 0.92138671875, -0.59326171875, 1.56640625, -0.642578125, 0.75439453125, 1.3681640625, -2.560546875, 1.7451171875, -0.431884765625, 1.0166015625, -1.3232421875, 0.7451171875, 0.033294677734375, -0.9677734375, 0.264404296875, 0.72412109375, -1.1220703125, -0.3388671875, -1.095703125, -0.56640625, -0.69970703125, -0.31591796875, -0.197998046875, -1.4541015625, 0.334716796875, 1.25, 0.381591796875, 0.2783203125, -1.013671875, 0.004573822021484375, -0.77978515625, -0.76513671875, 0.135986328125, 3.03125, -0.62646484375, 0.1328125, -0.65673828125, 2.447265625, 0.78955078125, -0.580078125, 1.3994140625, 0.1234130859375, -0.27001953125, -0.09698486328125, -2.71484375, -0.74365234375, 0.302734375, -0.59521484375, -0.63134765625, -0.015838623046875, 0.303955078125, -0.7666015625, -0.5615234375, 0.472412109375, -0.29345703125, -0.361328125, -0.87255859375, -0.9306640625, -1.62109375, -0.27099609375, -0.6728515625, 0.146240234375, -0.79638671875, -2.01171875, -0.56884765625, -0.439208984375, -0.6357421875, 0.53515625, -0.73681640625, 0.5400390625, -2.146484375, 0.669921875, 0.1336669921875, 0.3935546875, 0.004001617431640625, -0.615234375, -0.7109375, 2.328125, 0.04730224609375, 0.2337646484375, 0.546875, -1.689453125, -0.30419921875, -0.1387939453125, 0.91943359375, -1.076171875, -1.6953125, -0.210693359375, -2.2734375, 1.7998046875, 0.46142578125, -0.0172882080078125, -0.8564453125, -0.1468505859375, -1.748046875, 0.734375, -1.044921875, 0.0634765625, 0.42236328125, -0.4287109375, 1.0751953125, 0.4365234375, 1.6181640625, -0.1357421875, -0.43310546875, -0.8349609375, -0.1912841796875, -0.05999755859375, 0.81689453125, -1.1572265625, 0.057708740234375, -0.05084228515625, -1.7236328125, -0.1922607421875, 0.71923828125, 1.1669921875, 0.53662109375, -0.358154296875, 0.497314453125, 0.7265625, 0.884765625, -0.85546875, 0.71435546875, 0.1820068359375, -0.445068359375, -0.66552734375, 1.2529296875, -0.016510009765625, -1.166015625, 1.611328125, -0.87744140625, 0.00020945072174072266, 0.0018625259399414062, -1.259765625, 0.37744140625, 1.7197265625, 0.5322265625, 0.60546875, -0.34228515625, -0.95458984375, -0.630859375, -0.1612548828125, 0.76416015625, 0.0026416778564453125, 0.52685546875, -1.1298828125, -0.53662109375, -1.4677734375, 0.6083984375, -0.439208984375, -0.6025390625, -0.94384765625, 1.212890625, 0.1251220703125, 1.9736328125, -1.0908203125, -0.60302734375, -0.595703125, -0.8681640625, -1.04296875, 0.6337890625, 2.283203125, -0.8525390625, 1.1142578125, 0.73291015625, 0.64453125, -1.033203125, -0.12939453125, 0.3701171875, -0.55908203125, -0.0831298828125, -1.6103515625, -1.5400390625, -1.1875, 1.0068359375, 0.186279296875, -1.3857421875, -0.272216796875, -0.73974609375, 0.299560546875, 1.599609375, -1.552734375, -0.8837890625, 0.118408203125, 0.079833984375, -0.623046875, 0.44921875, -0.264404296875, 0.8193359375, -0.274658203125, -0.345947265625, -1.583984375, 1.97265625, -0.72607421875, -0.458251953125, 1.568359375, -0.07952880859375, -0.7177734375, -1.0126953125, 1.7978515625, 0.08331298828125, 0.296630859375, -0.58447265625, -0.002471923828125, -0.6689453125, -0.0218658447265625, 0.533203125, 0.1929931640625, 1.4775390625, -0.576171875, 0.337158203125, 0.274658203125, -0.0985107421875, 1.5537109375, 0.328125, -0.931640625, -0.08221435546875, -1.322265625, 0.57177734375, 0.12030029296875, -1.5732421875, -1.607421875, -0.27880859375, -0.09197998046875, -1.8486328125, 1.01171875, -1.1591796875, 0.1802978515625, 0.250732421875, -2.45703125, 0.044219970703125, -0.1378173828125, 1.3466796875, -0.5712890625, 0.189453125, 0.5986328125, 0.22802734375, -0.2255859375, 0.276123046875, -1.4609375, 0.10821533203125, -0.83837890625, -1.1064453125, 2.154296875, 0.80029296875, -1.375, -0.76513671875, 0.9951171875, 0.452392578125, -0.09796142578125, -1.876953125, -1.2158203125, 1.1767578125, 0.307861328125, -1.044921875, 0.62158203125, 0.283203125, 0.432373046875, 1.3828125, 0.11102294921875, 0.85595703125, 0.22705078125, 0.83642578125, 0.08038330078125, 0.09832763671875, 1.556640625, -0.50341796875, -1.1259765625, -0.8447265625, 0.78173828125, -0.0894775390625, 0.3955078125, 0.40576171875, 0.892578125, -0.419921875, -0.88916015625, -0.71826171875, 0.6064453125, -1.5830078125, 0.9482421875, 1.529296875, 1.1435546875, -1.4287109375, 0.016998291015625, -1.17578125, -0.8505859375, 0.54931640625, 0.049346923828125, 1.0341796875, 0.2327880859375, -0.74853515625, -0.2088623046875, -0.8271484375, -0.982421875, 0.54248046875, 1.1044921875, 0.131591796875, 0.12054443359375, 1.0654296875, 0.63427734375, -1.2763671875, -0.410888671875, -0.80224609375, 2.9609375, -0.1502685546875, 0.87109375, -0.100341796875, -0.86767578125, 0.3544921875, 1.716796875, -0.73291015625, -0.0701904296875, -1.095703125, -1.7626953125, 0.21337890625, -0.8837890625, -0.6015625, -0.65380859375, 1.328125, 0.94287109375, -0.94873046875, 0.1778564453125, 0.83203125, 0.040130615234375, -2.37890625, -0.94873046875, 1.052734375, -1.0390625, -2.6875, -0.241455078125, -0.18701171875, 1.0234375, -2.158203125, 1.591796875, 0.31787109375, -0.369873046875, 1.23046875, -0.1683349609375, 0.376220703125, -0.114501953125, 0.32763671875, -0.214111328125, 0.90283203125, -0.61865234375, 1.7626953125, 0.4423828125, -0.3447265625, -0.271484375, 2.14453125, 1.0166015625, 0.68310546875, -1.5224609375, -0.6865234375, -0.4599609375, 0.1685791015625, 1.314453125, 1.73828125, 1.5966796875, 0.16650390625, -1.2314453125, -1.580078125, -0.546875, 0.880859375, -0.81787109375, 2.39453125, -2.154296875, 0.95361328125, 0.1378173828125, -0.0823974609375, 1.54296875, 0.275146484375, 0.23193359375, 1.48046875, 0.5634765625, 0.052032470703125, 0.609375, -0.85107421875, -0.433349609375, -1.1669921875, -1.0478515625, 0.9521484375, 1.8134765625, -0.237060546875, -0.1796875, -0.374755859375, 1.2607421875, -0.1365966796875, -0.45556640625, 1.703125, -0.0271453857421875, 1.318359375, 0.5205078125, -0.71728515625, -0.271240234375, 1.3427734375, 1.009765625, 1.091796875, 0.2017822265625, -0.431396484375, -2.08984375, 0.2032470703125, -0.1932373046875, 0.501953125, -0.0162811279296875, -1.3896484375, -0.54150390625, 1.6572265625, 0.053436279296875, -0.6376953125, 0.50048828125, -0.3984375, -1.0634765625, 1.4384765625, 1.6337890625, 1.0654296875, -0.2052001953125, -0.95751953125, -1.0703125, -0.07537841796875, -0.59912109375, 1.083984375, 0.47119140625, -0.650390625, -2.3046875, -0.259521484375, -0.386474609375, 0.289306640625, 0.9228515625, 0.276611328125, 0.85986328125, -1.05859375, 0.050079345703125, -0.2403564453125, 0.471923828125, 0.449951171875, -0.206787109375, 0.4150390625, 1.64453125, -0.456787109375, -0.98876953125, 1.4189453125, -0.286865234375, 0.94970703125, -0.68408203125, -0.482177734375, -1.4951171875, -0.30078125, -2.189453125, -0.69091796875, 0.52880859375, -1.0732421875, 0.75, -1.0673828125, -0.49267578125, -0.31982421875, 0.65478515625, 0.927734375, -0.1368408203125, 0.473388671875, 0.3046875, -0.156494140625, -1.583984375, -1.705078125, 0.615234375, 0.728515625, -2.17578125, 0.951171875, -0.634765625, 1.50390625, 1.4912109375, -2.447265625, 1.443359375, -0.65966796875, 0.5322265625, 0.1156005859375, -2.4296875, -0.24609375, -0.67822265625, 1.8076171875, -1.0849609375, -0.49267578125, -1.244140625, -0.28564453125, -0.53076171875, 0.89697265625, 0.00696563720703125, 1.7265625, 0.96435546875, 1.080078125, -1.6181640625, -3.33203125, -0.57470703125, 0.01027679443359375, 0.231201171875, -1.388671875, 0.59033203125, -0.50830078125, 0.31884765625, -0.576171875, -0.2734375, -0.80078125, 1.1904296875, 1.6357421875, 0.2091064453125, -0.155517578125, -0.85986328125, 0.57958984375, -1.0498046875, 2.306640625, -1.4052734375, -0.85498046875, -0.71142578125, 0.2998046875, -0.65478515625, 1.802734375, 1.041015625, 0.56103515625, 1.1103515625, -0.521484375, -2.412109375, 1.455078125, -1.1318359375, 0.26708984375, 0.88330078125, -1.8212890625, 0.01169586181640625, 1.1181640625, 0.90771484375, -0.7939453125, 0.169189453125, -0.041717529296875, -1.259765625, -1.73828125, -1.53125, -0.87841796875, -1.0087890625, -0.27197265625, -0.845703125, 0.673828125, 0.59228515625, -0.425048828125, 1.2978515625, -0.7666015625, 0.720703125, -0.067626953125, 0.42236328125, -1.634765625, -1.056640625, -0.04022216796875, 0.00524139404296875, -0.39990234375, 1.2099609375, 0.60107421875, -0.1937255859375, -1.0380859375, 1.7333984375, 0.0126190185546875, 0.387451171875, 0.54150390625, 0.10888671875, -1.0703125, -1.8916015625, -1.2724609375, -0.126220703125, -1.130859375, -1.177734375, -1.6357421875, 0.2491455078125, -1.154296875, 0.270263671875, 1.1328125, 0.68115234375, 1.2060546875, 0.328369140625, -0.75, -1.1220703125, -1.328125, -0.931640625, 1.625, -0.9404296875, 1.212890625, -0.253173828125, -0.9384765625, -0.34375, -0.98291015625, -0.51318359375, 1.7548828125, -1.8935546875, -1.9462890625, 1.1474609375, -0.01425933837890625, 1.087890625, 0.496337890625, -1.21484375, -0.1275634765625, 2.228515625, 2.8125, 0.62255859375, -0.287841796875, 1.1328125, 0.79638671875, 2.140625, 0.98779296875, 0.72900390625, 0.01678466796875, -0.09368896484375, 0.6923828125, 1.83203125, -1.6474609375, -0.7041015625, 0.23876953125, -1.583984375, 0.87548828125, 1.6259765625, -2.076171875, -0.1771240234375, 0.295166015625, -0.00130462646484375, 0.11334228515625, -1.173828125, 0.322509765625, 1.4072265625, 0.830078125, -0.58642578125, -0.783203125, 0.287841796875, -0.2213134765625, -1.5400390625, 0.88623046875, 0.048553466796875, -0.148193359375, 0.300048828125, 1.4228515625, 0.27880859375, 1.9609375, -1.341796875, -0.70458984375, 0.2408447265625, 0.73388671875, -0.061187744140625, 1.4169921875, 0.58154296875, -0.76953125, 1.9609375, 1.3466796875, 0.045013427734375, 1.4833984375, -0.305419921875, -1.1513671875, 0.7431640625, -1.0751953125, -0.2166748046875, 1.4345703125, -1.7119140625, 1.1640625, -1.7607421875, -0.82421875, -0.52685546875, 0.73583984375, -0.96875, -0.47607421875, -0.8984375, 0.377197265625, 0.196533203125, 0.387451171875, 0.00917816162109375, 0.336669921875, -0.0921630859375, 0.48681640625, -0.82421875, 0.90234375, -0.84716796875, 1.0048828125, 2.1015625, 0.1710205078125, -1.412109375, 1.361328125, -0.78955078125, 1.8212890625, 0.76806640625, 1.4267578125, -0.521484375, -0.07611083984375, -0.9033203125, -0.5615234375, -1.1533203125, -0.0672607421875, 1.26953125, 1.3291015625, -0.6162109375, 0.55224609375, -0.174560546875, 1.0576171875, 1.6220703125, 1.453125, -0.517578125, -0.46240234375, 0.20849609375, -1.4208984375, 0.81103515625, -0.9326171875, -0.798828125, 1.359375, 0.257080078125, -1.02734375, 0.3681640625, 0.87109375, 0.79150390625, -1.75390625, 0.345458984375, 0.8623046875, -1.21875, -1.3486328125, 0.2425537109375, -0.46826171875, 0.7978515625, 0.765625, 1.5849609375, 1.2744140625, -0.36474609375, -0.68701171875, 2.44921875, 2.390625, 0.75048828125, -0.81884765625, -1.1923828125, 1.4453125, -1.2138671875, 0.96826171875, 0.88330078125, -0.154296875, -1.6650390625, -0.60595703125, 1.005859375, 0.8017578125, -0.10882568359375, -1.265625, -0.65966796875, 0.8720703125, 0.432373046875, -0.352294921875, 0.4423828125, -1.0908203125, -0.1644287109375, -0.2149658203125, -0.048431396484375, 1.7529296875, -0.36279296875, -1.2646484375, -1.2392578125, 2.529296875, 0.7001953125, 0.84326171875, 0.10906982421875, -0.6474609375, -0.251708984375, 1.26953125, -0.5830078125, 0.6943359375, -0.02191162109375, -1.5849609375, -0.8505859375, -1.7109375, -2.66015625, 1.4775390625, 2.44921875, 0.358642578125, 0.66162109375, -1.595703125, 1.4267578125, 0.136962890625, 0.4033203125, -0.209228515625, -1.8564453125, -1.095703125, -0.365478515625, -0.560546875, 0.228759765625, 1.107421875, -0.4033203125, -1.9833984375, 0.791015625, -0.1285400390625, 0.0960693359375, 1.19140625, 0.4267578125, -2.087890625, 0.73974609375, -0.138671875, 0.0018320083618164062, 0.47412109375, 1.6416015625, 0.89306640625, 2.384765625, -1.4619140625, 0.587890625, 0.11492919921875, 0.317138671875, -0.00238800048828125, 0.56640625, -0.48828125, -1.359375, -1.28125, 0.471435546875, -1.2001953125, -0.441650390625, 1.658203125, 1.5546875, 1.2578125, -0.130859375, -0.39794921875, 0.101806640625, 0.6279296875, -0.94091796875, 1.7861328125, 2.166015625, 1.4912109375, 0.2802734375, -0.7978515625, 0.061370849609375, 0.4716796875, 0.131103515625, 1.8955078125, 0.5595703125, 1.48046875, -1.001953125, -1.7822265625, 1.3447265625, -0.403076171875, -0.0185089111328125, 0.042083740234375, 0.775390625, -0.152099609375, -0.41845703125, -1.75, 0.61083984375, 0.0301666259765625, 1.140625, 0.794921875, -0.49609375, -1.9619140625, -0.37890625, 0.6845703125, 1.4501953125, 0.09332275390625, 0.96435546875, 0.32763671875, 0.394775390625, -0.32958984375, 0.52490234375, 0.50341796875, -0.4677734375, -0.293212890625, -0.70654296875, -2.287109375, 0.0736083984375, -1.6865234375, -1.177734375, -0.54833984375, -0.039306640625, 0.99951171875, -0.487548828125, -0.495849609375, 0.0625, -2.23828125, -1.01171875, 0.2298583984375, -0.1768798828125, -0.468994140625, 0.301513671875, -1.380859375, 0.65673828125, 0.49462890625, 0.1510009765625, -2.5390625, 0.143310546875, -0.83154296875, -0.61572265625, -0.95458984375, 1.056640625, 0.34912109375, 0.1715087890625, 0.68896484375, 0.984375, -0.2144775390625, -0.580078125, -0.0535888671875, -0.142578125, -0.3330078125, 0.431884765625, -1.251953125, -0.487060546875, 1.2041015625, 0.84912109375, 0.916015625, 0.33251953125, -1.8994140625, -0.7880859375, 0.79541015625, -0.87939453125, 0.268798828125, -1.5361328125, 0.0202484130859375, 0.6044921875, 0.11297607421875, 0.3427734375, -1.4677734375, 0.697265625, -0.5634765625, -0.5234375, -1.1328125, -0.69677734375, 2.275390625, 1.2705078125, 0.11865234375, -0.391845703125, -1.3818359375, -1.7119140625, -1.5947265625, 0.429443359375, 2.04296875, 0.107421875, 0.2010498046875, 0.7744140625, -0.429443359375, 0.421142578125, -1.2744140625, 0.74365234375, -0.6962890625, 1.4921875, -0.349365234375, 1.9140625, 0.275634765625, -0.178466796875, -0.252197265625, 0.0633544921875, 0.20556640625, 1.0908203125, -1.244140625, -0.46728515625, -0.802734375, -0.552734375, -0.402587890625, -1.052734375, -0.1966552734375, -0.3486328125, 0.7255859375, 0.2371826171875, 1.033203125, 0.30224609375, 2.142578125, -3.142578125, 0.060455322265625, -0.25146484375, -0.305908203125, -0.29638671875, -0.5634765625, -0.85009765625, 1.095703125, -0.93310546875, -0.8193359375, -1.8359375, 0.30224609375, -1.576171875, -0.97998046875, 0.4423828125, 1.04296875, -0.99658203125, -0.1053466796875, 0.1888427734375, -0.34033203125, 0.2044677734375, -0.8115234375, -0.09771728515625, 1.3828125, 0.1837158203125, -0.66650390625, -0.52783203125, 0.44580078125, -0.4951171875, 0.8740234375, -1.3984375, -0.31982421875, 1.083984375, 0.974609375, 0.806640625, -1.181640625, -1.4365234375, 0.408203125, 1.1943359375, 1.1767578125, -1.779296875, -0.0252838134765625, 0.32373046875, -0.67333984375, 0.06884765625, -0.2078857421875, 0.970703125, -0.71826171875, 1.11328125, -0.0673828125, -1.1123046875, -1.232421875, 2.3515625, 0.403564453125, -0.53515625, -0.0792236328125, 0.418212890625, -0.162841796875, -0.6494140625, -1.48828125, -1.3017578125, 0.7890625, 1.8974609375, -0.57177734375, 1.0244140625, 0.051177978515625, 0.344482421875, 0.190185546875, -0.8310546875, 0.054046630859375, -1.7255859375, 0.2012939453125, -1.20703125, -0.86962890625, -0.6826171875, -0.5400390625, 1.0146484375, -0.09619140625, 0.7744140625, 1.2470703125, -0.61865234375, -0.6943359375, -1.58203125, -1.064453125, -0.69921875, 1.0849609375, 1.3388671875, 1.2626953125, 1.587890625, 1.16796875, -0.06439208984375, -0.1656494140625, -1.4873046875, -0.470703125, -1.9306640625, -2.03515625, -0.383056640625, -0.7265625, -0.246826171875, 1.01171875, -0.5927734375, 0.042938232421875, 0.470458984375, -0.8134765625, -0.442138671875, 0.560546875, -0.446044921875, 0.486083984375, 0.0352783203125, 0.6142578125, 0.65234375, -1.0029296875, -1.5908203125, 0.75634765625, 0.6083984375, -0.841796875, 1.8935546875, 0.470703125, -1.3056640625, 0.8466796875, -0.36474609375, -1.6552734375, -1.09375, 0.26220703125, 0.0682373046875, 0.0740966796875, -0.90234375, 0.365966796875, -0.1341552734375, -0.0992431640625, -1.662109375, -0.8662109375, -1.0537109375, 0.328857421875, -0.05767822265625, 0.76171875, -2.009765625, -0.1163330078125, -0.400634765625, -0.41455078125, -1.712890625, -2.8828125, -0.48876953125, -0.29296875, 1.3642578125, 0.42138671875, -0.0082244873046875, 1.0107421875, -1.673828125, 1.06640625, 1.228515625, 1.3564453125, 0.074462890625, -0.419189453125, 0.380126953125, -1.4453125, -1.16015625, -1.2421875, 0.204833984375, 0.93212890625, -1.73046875, -0.6337890625, -0.1629638671875, -0.310302734375, 0.10552978515625, -0.1539306640625, 0.6044921875, 1.8642578125, -0.72705078125, -0.43017578125, 0.1612548828125, -0.08551025390625, -0.251220703125, -2.00390625, -0.52978515625, 0.70654296875, -0.2379150390625, 1.9189453125, -1.9931640625, 0.16552734375, 0.48681640625, 0.1278076171875, -0.268798828125, -0.1971435546875, 1.6484375, -0.9169921875, -1.5234375, -0.9482421875, 0.057708740234375, 0.1715087890625, 0.34423828125, -0.267578125, -0.300537109375, -0.17041015625, -0.5009765625, -0.1114501953125, -1.458984375, -1.228515625, 0.74755859375, -0.7724609375, 0.452880859375, 0.72412109375, -0.52685546875, 0.72900390625, 1.064453125, -0.89794921875, 1.8583984375, 0.128173828125, 0.04071044921875, 0.0309600830078125, -2.892578125, 0.99609375, -0.48046875, 0.308837890625, -1.9150390625, -0.1580810546875, 0.152099609375, -0.8818359375, -0.10333251953125, 0.82470703125, 0.26904296875, -1.1640625, -0.0154266357421875, 0.31982421875, 0.5810546875, -0.2042236328125, 0.88720703125, -0.2841796875, 0.97802734375, 0.673828125, -0.44580078125, 1.05078125, 2.42578125, 0.16357421875, 1.806640625, 0.322021484375, 0.625, 1.9609375, 0.97998046875, 1.0732421875, 2.404296875, 1.2578125, 1.0966796875, -0.99267578125, 1.4013671875, 0.78515625, 2.31640625, -1.0107421875, -2.255859375, 0.146728515625, 1.1240234375, 0.33251953125, -0.82666015625, -0.1044921875, 0.2392578125, -2.65234375, 0.57568359375, -0.50537109375, -1.4345703125, 1.2265625, -0.74072265625, -0.1685791015625, -1.4423828125, -1.416015625, 2.380859375, -1.1806640625, -0.58740234375, 0.705078125, 0.121337890625, 0.123779296875, 0.521484375, 0.8330078125, 0.38330078125, -0.5888671875, -0.580078125, 1.25390625, 0.0171051025390625, 2.7109375, -0.66845703125, 1.515625, -0.348388671875, -0.454345703125, -1.2685546875, 0.89453125, 0.2587890625, -0.059722900390625, -0.623046875, 0.1551513671875, -0.1552734375, -0.04986572265625, -2.017578125, -0.1595458984375, -0.1671142578125, -0.11383056640625, 0.62060546875, -2.177734375, -1.2578125, 0.46923828125, 0.6474609375, -0.1695556640625, 0.568359375, 2.1796875, -1.1728515625, 0.251220703125, -0.56494140625, -2.06640625, -0.313232421875, 1.0400390625, -1.0224609375, 0.5361328125, 1.0029296875, 2.201171875, -1.1728515625, 0.888671875, 0.57080078125, 0.81787109375, -0.59033203125, 0.350830078125, 0.365234375, 0.70556640625, -1.6796875, 1.1552734375, 0.62646484375, 1.740234375, 1.025390625, 0.51318359375, 0.63916015625, -0.87451171875, 1.1005859375, 0.404052734375, 1.005859375, 0.82861328125, -0.5595703125, -1.7353515625, 2.09375, -0.85693359375, -1.287109375, -0.9765625, 0.87841796875, 0.984375, 1.1533203125, 0.020172119140625, -0.623046875, -0.7060546875, -0.587890625, -1.25390625, -0.341552734375, -1.6904296875, 0.1702880859375, -1.9208984375, 0.2218017578125, 0.6806640625, -0.87109375, 1.580078125, 0.327392578125, -1.443359375, -1.9091796875, 0.0787353515625, 0.9228515625, -0.52099609375, -1.0771484375, 0.65283203125, 1.837890625, 0.66064453125, 0.8603515625, -0.75048828125, -0.431396484375, 0.69970703125, 0.626953125, 0.77734375, -0.552734375, 0.45361328125, -1.8212890625, -0.10540771484375, -0.1307373046875, 0.237060546875, -0.1253662109375, 0.66796875, -0.2154541015625, -1.216796875, 0.453857421875, 0.07177734375, 1.333984375, -0.2578125, -0.68115234375, -0.173828125, 0.7578125, -1.25390625, 1.8408203125, -0.6220703125, 0.344970703125, 1.888671875, 0.6806640625, -0.6962890625, -1.4296875, 0.0482177734375, -0.038299560546875, -0.05230712890625, -0.59423828125, 0.68798828125, 0.6103515625, -0.0897216796875, 0.3798828125, 1.529296875, 0.279052734375, -1.2392578125, -0.88037109375, -0.00894927978515625, -0.4375, -0.298583984375, -0.52099609375, 1.0107421875, 0.6689453125, -0.77392578125, -0.06842041015625, 2.333984375, -0.01953125, -0.666015625, -1.93359375, 1.025390625, -0.37548828125, 0.29052734375, -1.7236328125, 0.56103515625, 1.64453125, -0.96923828125, -0.159423828125, 1.1826171875, -1.509765625, -0.54931640625, -0.97802734375, 0.751953125, 1.9453125, -0.328369140625, -0.09124755859375, -0.6982421875, -0.76123046875, 1.19921875, -1.0673828125, -0.323486328125, 2.048828125, 0.8662109375, 1.73828125, 0.765625, 0.6201171875, -0.7802734375, -0.45263671875, -1.134765625, 0.66845703125, -0.91162109375, 1.5263671875, -1.4833984375, 0.99853515625, 2.486328125, 1.671875, -1.4580078125, -2.001953125, 0.541015625, 0.859375, -0.9013671875, -0.962890625, 1.26953125, -0.3896484375, -1.400390625, -0.54296875, 0.12103271484375, -0.533203125, -0.26220703125, -0.51513671875, 0.114990234375, 1.68359375, 0.5126953125, 1.2783203125, 0.7099609375, -0.62744140625, -0.053009033203125, 0.64599609375, 0.91162109375, 1.34375, -0.67041015625, 1.087890625, -0.266845703125, 0.173583984375, -2.025390625, 0.92578125, 0.217529296875, -0.138916015625, -1.96484375, 0.9384765625, 0.58251953125, 0.87841796875, 1.076171875, 1.7412109375, 0.358154296875, 2.8125, -0.94873046875, 0.68359375, 0.3427734375, 0.26904296875, -0.5546875, 0.467529296875, -0.8447265625, 1.345703125, -2.142578125, -0.79931640625, 0.0880126953125, 1.61328125, -0.70556640625, 1.2197265625, -1.4287109375, 0.05706787109375, -0.1346435546875, -0.4443359375, -1.8046875, 0.4990234375, 0.2025146484375, 0.81103515625, -0.650390625, 0.170166015625, -0.383056640625, -0.857421875, 0.343017578125, -0.66455078125, -0.72705078125, 0.10650634765625, 1.390625, -0.55126953125, 0.5791015625, -0.73828125, -0.436279296875, 1.1572265625, 0.9912109375, -0.37890625, 0.08013916015625, -1.54296875, -0.0301513671875, 1.0068359375, 0.1949462890625, -0.7841796875, 0.5888671875, 1.10546875, -0.7470703125, -0.9951171875, -1.271484375, -1.03515625, -1.99609375, -1.357421875, 0.61279296875, 0.8671875, 1.298828125, -0.61279296875, 0.05841064453125, 1.08203125, 0.92822265625, 0.282958984375, 0.75927734375, 0.94873046875, -1.0849609375, 0.697265625, 0.68408203125, 1.7021484375, -0.4501953125, -0.20263671875, -1.0576171875, 0.72021484375, 0.005275726318359375, 0.94384765625, 0.80712890625, -1.4931640625, 0.1866455078125, -0.444580078125, -0.96533203125, -0.463623046875, -0.367431640625, 1.0478515625, 0.85791015625, 0.51123046875, 1.03125, 1.8828125, -0.2259521484375, 0.03887939453125, 0.80322265625, 0.5166015625, -0.43212890625, 1.6044921875, -0.049041748046875, 0.1666259765625, 0.383544921875, 0.6884765625, 1.7060546875, -0.061981201171875, -0.8779296875, -1.234375, 1.0009765625, -1.126953125, 0.2626953125, -0.88671875, -0.12249755859375, 0.325439453125, 0.31591796875, -0.357421875, -2.80078125, -1.9580078125, -1.3232421875, 0.33837890625, 1.2470703125, 0.18017578125, -0.12030029296875, 0.74072265625, -0.0595703125, -1.029296875, 1.1591796875, 1.0576171875, -0.5869140625, -0.30810546875, -1.0390625, 0.9765625, 0.6962890625, 0.69140625, 0.51123046875, 1.9345703125, 0.5927734375, -0.865234375, -1.5009765625, 0.36328125, 0.435791015625, -0.8212890625, 1.7392578125, -0.448486328125, -1.798828125, 0.11297607421875, -0.1602783203125, -0.9619140625, 1.3359375, 0.5126953125, -0.339111328125, 0.1143798828125, -0.277099609375, 0.23583984375, 0.54736328125, 1.0029296875, 0.31298828125, -0.58349609375, 0.1656494140625, -1.3603515625, 1.5361328125, -1.4814453125, 2.27734375, 0.1904296875, -0.03253173828125, 1.2158203125, 0.4189453125, -2.751953125, -0.328369140625, -0.033203125, 1.443359375, 1.896484375, -1.060546875, 0.00666046142578125, -0.07171630859375, -1.189453125, -1.4677734375, 1.4921875, -1.40234375, -0.428466796875, -2.146484375, -2.125, 0.857421875, 0.09893798828125, -1.076171875, 1.169921875, 2.166015625, 0.00478363037109375, 1.3583984375, 0.07403564453125, 0.10345458984375, -0.032012939453125, -0.2275390625, 0.29931640625, -0.73828125, -1.357421875, -0.451171875, 0.90234375, 1.0478515625, 1.9521484375, -0.478515625, -0.345703125, 0.70849609375, 0.4375, 0.452392578125, 0.5947265625, 0.708984375, -0.67333984375, 0.048370361328125, 0.95703125, 0.66064453125, 0.384765625, -0.09423828125, 0.904296875, 0.229736328125, -0.37353515625, -0.06573486328125, 2.35546875, -0.4189453125, 0.5234375, -1.0205078125, -1.9697265625, -0.057220458984375, -0.5400390625, -1.4013671875, -1.4013671875, 0.355712890625, 0.50830078125, -0.1273193359375, 0.037872314453125, -0.322265625, 1.00390625, 0.80224609375, -1.53515625, -0.31787109375, 0.5810546875, 0.6201171875, -0.720703125, -0.39501953125, -0.75244140625, 1.3896484375, 0.904296875, -0.18505859375, -2.220703125, 1.298828125, 0.58740234375, 0.298095703125, -0.1865234375, 2.2265625, 0.06829833984375, -0.35205078125, 1.775390625, -0.2022705078125, 1.763671875, -0.421875, 0.1639404296875, 0.417724609375, -1.0654296875, -0.00440216064453125, -0.253173828125, 0.99267578125, 1.0068359375, 0.414306640625, -0.10260009765625, -1.4453125, -0.91650390625, -0.2095947265625, 0.271240234375, -0.7939453125, -0.66796875, -0.04827880859375, -1.7890625, 1.16796875, 1.2412109375, 1.62109375, -1.2138671875, 0.6796875, -0.1495361328125, -0.6533203125, 0.25537109375, -0.705078125, -1.6669921875, 1.5947265625, -0.2288818359375, -0.85888671875, 0.73828125, 0.266357421875, -0.37109375, -1.79296875, -0.50244140625, 0.2122802734375, -1.712890625, -0.08050537109375, -1.8876953125, 0.8857421875, -0.751953125, -1.3232421875, 0.54345703125, -0.61669921875, -0.497802734375, 0.61962890625, -0.0347900390625, -0.78076171875, 0.76123046875, -0.08013916015625, 1.3720703125, -0.71484375, 0.20166015625, 1.5595703125, 0.7763671875, 1.736328125, 1.712890625, 1.09375, -0.24072265625, 0.0711669921875, -1.44921875, -0.286376953125, -1.2568359375, 1.53125, -3.06640625, 0.5732421875, -1.4833984375, -0.13037109375, 0.9013671875, -0.70263671875, -2.51171875, -0.1898193359375, 1.8232421875, 0.3671875, -1.80078125, -0.67529296875, 0.43017578125, 0.6259765625, -0.60302734375, -0.7197265625, 0.2296142578125, -0.52392578125, -0.92724609375, -1.2451171875, -1.1123046875, -1.0537109375, -0.25390625, 0.43212890625, 1.2626953125, -0.98828125, -1.333984375, 0.7392578125, 0.6044921875, 0.1502685546875, 0.2939453125, 0.02734375, 0.853515625, -1.3955078125, 1.6845703125, 0.814453125, 2.193359375, -0.1373291015625, 0.0924072265625, 1.134765625, 0.28125, -1.24609375, -1.671875, -0.485107421875, -1.3076171875, -1.833984375, -0.853515625, 0.7275390625, -1.0546875, -0.7734375, 2.251953125, -1.06640625, -0.53369140625, 1.0537109375, -1.6943359375, -1.0, 0.54833984375, -1.962890625, -0.802734375, 0.93115234375, 0.362548828125, -0.468505859375, 0.28369140625, -1.078125, 0.994140625, -0.43994140625, 0.1788330078125, -1.560546875, -1.0048828125, -0.33935546875, -0.89306640625, 0.335205078125, -1.021484375, 0.767578125, 0.370361328125, -0.90966796875, -0.8095703125, 0.333251953125, 0.437255859375, -0.52783203125, -0.9697265625, -0.91552734375, -1.3369140625, 1.37109375, -0.29345703125, -0.81298828125, 0.56298828125, -1.244140625, 0.368896484375, 0.61376953125, -0.257568359375, 0.2578125, -1.0048828125, -0.6748046875, -0.57958984375, 0.36474609375, -0.44287109375, 0.307861328125, 1.150390625, -1.0673828125, -0.0101776123046875, -2.15234375, -1.7265625, 0.253662109375, -0.73779296875, 0.66796875, -0.79052734375, -1.5341796875, 1.75390625, -0.425537109375, -0.9287109375, -0.71728515625, -0.779296875, 0.419677734375, 0.34716796875, -1.3916015625, -0.51806640625, -0.461669921875, 1.880859375, -0.09381103515625, 1.1064453125, -0.021240234375, -0.681640625, -0.53662109375, 0.285888671875, 0.267333984375, -0.341796875, -0.005828857421875, 0.99560546875, -0.88134765625, 0.603515625, -0.82666015625, 0.48046875, -0.273681640625, -0.042816162109375, 0.6015625, -0.1865234375, 0.43408203125, 0.708984375, 0.4814453125, 0.364501953125, 0.560546875, 1.380859375, 0.5556640625, 2.181640625, 0.68896484375, -1.0458984375, 0.16162109375, -0.98486328125, -0.491943359375, -0.407470703125, -0.049163818359375, 0.004955291748046875, 0.59375, -0.334716796875, -1.3349609375, 0.373291015625, 0.677734375, 1.109375, -0.609375, 0.1884765625, -0.2088623046875, 0.65087890625, -0.408935546875, 0.94189453125, 0.91455078125, -1.8359375, -0.685546875, -1.5732421875, -1.2890625, -0.71533203125, -0.384765625, -0.62744140625, 0.64794921875, 0.9736328125, 0.418212890625, -0.67578125, 0.73486328125, -0.006786346435546875, 1.1884765625, 1.96875, -0.8154296875, -1.087890625, 0.62060546875, 0.66845703125, 1.0283203125, 0.046295166015625, 0.94775390625, 0.98876953125, 0.50439453125, -1.1484375, 1.4482421875, -0.07000732421875, -0.2056884765625, 0.321044921875, 1.3525390625, 0.0921630859375, -0.73876953125, -1.640625, -0.462646484375, -0.69775390625, 0.26416015625, -2.271484375, 0.76025390625, 0.95849609375, 2.48046875, -0.401611328125, -0.529296875, 1.5791015625, -0.55078125, 0.472412109375, 0.10650634765625, 0.420166015625, -0.35400390625, -1.212890625, -0.1402587890625, 0.320068359375, -2.189453125, -0.61328125, -0.1282958984375, 0.5478515625, 1.5205078125, -0.45361328125, 0.53662109375, 0.75732421875, 0.6572265625, -1.6181640625, 0.0009984970092773438, -1.169921875, 1.158203125, -1.712890625, 1.240234375, -0.73583984375, -0.1549072265625, 1.1708984375, -1.873046875, -2.685546875, 0.40283203125, 0.33984375, 0.383544921875, -0.202880859375, -0.57080078125, -0.3525390625, 0.9609375, -1.0859375, 0.150390625, -1.5966796875, -0.414794921875, 0.35595703125, -0.12939453125, -0.3154296875, 0.384033203125, -0.2266845703125, 0.019927978515625, -0.336669921875, 0.328369140625, 1.041015625, 0.0927734375, 0.62744140625, -0.65380859375, -0.416259765625, 0.384033203125, -0.366943359375, -0.1376953125, -0.1334228515625, 0.5341796875, -0.72900390625, 1.259765625, 0.386962890625, 0.36328125, 1.494140625, -0.2474365234375, 1.4189453125, -0.62109375, -1.5166015625, 1.9091796875, -0.2442626953125, 0.01007080078125, 0.0963134765625, 0.89697265625, -0.035797119140625, -0.23974609375, -0.31640625, -0.1683349609375, 0.73291015625, 0.34326171875, -0.334228515625, 0.414794921875, -0.77783203125, -0.51513671875, -0.955078125, 0.81982421875, 0.0142364501953125, 0.07421875, -0.8330078125, 0.408203125, 0.399169921875, -0.873046875, 0.521484375, 0.517578125, 0.099609375, 2.546875, -0.258544921875, 1.7998046875, -1.1865234375, 1.1728515625, -1.763671875, -1.2138671875, 0.67626953125, -0.8515625, -0.3974609375, 0.495361328125, 0.1544189453125, -1.8388671875, 0.9111328125, -0.6884765625, 1.9560546875, -0.63134765625, -0.349853515625, -0.0282745361328125, -0.77001953125, -0.861328125, -0.5419921875, -0.2039794921875, 0.00975799560546875, 0.5693359375, -1.5244140625, -0.283203125, 0.91455078125, 2.0546875, -1.513671875, 0.65087890625, -0.62744140625, 1.4033203125, -0.43310546875, 0.6015625, 0.2401123046875, -0.2177734375, -2.001953125, 0.6767578125, 0.451416015625, -1.7998046875, -0.471435546875, 0.58642578125, -1.0830078125, -1.8994140625, 0.20947265625, 1.3154296875, 0.35400390625, -0.2431640625, 1.3056640625, -0.67041015625, 1.6142578125, -1.685546875, 1.07421875, 0.8759765625, -0.34716796875, -0.49365234375, 0.056060791015625, 0.98388671875, -0.148193359375, -0.2275390625, 0.00506591796875, 1.77734375, 0.1824951171875, -1.7177734375, 0.06768798828125, 0.375244140625, -0.22900390625, -0.60595703125, -0.0061492919921875, 0.931640625, 0.346923828125, -1.453125, -0.81298828125, 0.388916015625, -0.66015625, 0.84228515625, 0.2470703125, -0.76220703125, -0.478271484375, -2.318359375, -0.6494140625, 1.970703125, -0.3056640625, -0.267578125, -0.0022411346435546875, 0.040283203125, -2.041015625, 0.07623291015625, 1.6572265625, 0.8662109375, 0.80810546875, -0.022796630859375, 0.96240234375, 1.7705078125, 0.73828125, -0.206298828125, 2.052734375, -0.5537109375, -0.53515625, 0.5830078125, -2.236328125, -1.1435546875, -1.4140625, -0.3349609375, -0.90966796875, -0.7080078125, -0.166259765625, 0.35986328125, 0.1614990234375, -0.5576171875, -0.67578125, -0.039337158203125, 1.2412109375, 0.12359619140625, 1.228515625, -1.18359375, -1.8603515625, -1.7626953125, 0.472900390625, -1.4853515625, -0.47998046875, -0.09490966796875, -1.7685546875, -0.267578125, 0.96240234375, -0.1121826171875, 1.0693359375, -1.2998046875, 0.1932373046875, -0.394287109375, 1.4296875, 0.491455078125, 2.705078125, -1.3583984375, -2.255859375, -1.9453125, 0.419677734375, 1.3623046875, -1.044921875, 1.0302734375, 0.97265625, -0.5419921875, 1.34375, 1.3837890625, 0.6201171875, -0.96240234375, -0.27734375, -0.1529541015625, -0.716796875, 2.203125, -0.5517578125, 0.10894775390625, 0.64794921875, 0.279541015625, -0.61962890625, 0.04815673828125, 0.0997314453125, -0.84326171875, -0.28076171875, 0.3740234375, 0.1844482421875, 1.2099609375, -0.83740234375, 0.11273193359375, 0.41552734375, -1.357421875, 1.6513671875, -0.88232421875, 0.65576171875, -0.59912109375, -1.728515625, 0.560546875, 0.460205078125, 0.00870513916015625, 0.57666015625, -1.0146484375, 0.6943359375, 1.296875, 0.99365234375, -0.66552734375, 0.353759765625, -1.3388671875, 0.0831298828125, 0.430419921875, -0.231689453125, -1.26171875, -0.477294921875, 0.318115234375, -0.81298828125, 0.361083984375, 1.2099609375, 1.119140625, 2.52734375, -0.2174072265625, -0.34033203125, 0.724609375, 1.4619140625, -0.51708984375, -0.64404296875, -0.437255859375, -0.10986328125, 0.7412109375, -0.458984375, 0.1002197265625, 0.6083984375, -0.0399169921875, -0.8330078125, 1.0224609375, 0.412353515625, -0.662109375, 0.31640625, -0.76318359375, 0.2020263671875, 2.60546875, 1.71875, 1.427734375, -0.9697265625, 0.0968017578125, 1.619140625, -1.0576171875, -1.1826171875, 0.94970703125, 0.51025390625, -1.716796875, 2.044921875, 0.7626953125, -1.18359375, -1.1591796875, 0.74169921875, 1.2431640625, 1.0625, 1.3154296875, -2.69921875, -0.237548828125, 0.387451171875, 0.61962890625, -0.66650390625, -0.59326171875, 0.33642578125, -0.26513671875, 1.177734375, -1.6142578125, -0.40869140625, 0.70458984375, 0.399658203125, 0.2054443359375, 2.583984375, -2.009765625, -0.72119140625, -0.77099609375, -0.3125, -0.0062408447265625, 1.18359375, -0.303466796875, -0.16552734375, -2.552734375, 0.78759765625, -1.166015625, -1.244140625, 2.267578125, 1.2763671875, -0.08343505859375, 0.97265625, 1.576171875, -2.5625, -1.4228515625, -0.5458984375, -0.537109375, 0.86376953125, -0.44775390625, -1.447265625, -1.8818359375, 0.059234619140625, 0.53564453125, -0.0237274169921875, -0.6025390625, 1.017578125, -1.5400390625, -0.451416015625, -1.2138671875, 0.457275390625, -0.2138671875, 0.392822265625, 0.94384765625, 0.80224609375, 0.67919921875, 0.29443359375, -0.486083984375, -1.3291015625, -0.386962890625, 1.2001953125, -0.947265625, 0.14453125, -0.5234375, 0.280029296875, -0.21630859375, 1.435546875, 1.08984375, -0.62646484375, 1.4228515625, 0.8212890625, 1.2734375, 0.275146484375, -0.338134765625, 1.4853515625, 0.1395263671875, 0.01291656494140625, 1.35546875, 1.6025390625, 1.123046875, -0.44873046875, 0.876953125, -0.55859375, -0.83740234375, -0.189208984375, 0.318115234375, -0.06591796875, 0.302490234375, 1.55859375, -1.1689453125, -1.3408203125, 1.0908203125, -1.4453125, 1.4306640625, 0.669921875, 1.1220703125, 1.8203125, 0.65673828125, -0.016326904296875, -0.5068359375, 1.1171875, -0.8916015625, -3.1171875, -0.609375, -1.080078125, 1.3310546875, 1.1044921875, -0.44091796875, 0.309326171875, 0.55322265625, 0.225341796875, -0.681640625, 1.21875, 0.2998046875, 0.2568359375, -0.607421875, -1.552734375, -1.0634765625, 1.515625, -0.1959228515625, -0.51806640625, -1.416015625, -0.8955078125, 0.0513916015625, -0.1517333984375, -0.40625, -0.7294921875, -0.72314453125, -0.251953125, -0.9033203125, -0.0065460205078125, 0.759765625, 2.072265625, -0.9453125, 0.5, 0.88037109375, -1.7353515625, 0.039215087890625, -0.5986328125, -1.0576171875, -0.032958984375, 1.29296875, -1.107421875, 1.359375, -0.21484375, 1.0166015625, -1.6123046875, 1.4736328125, -1.3046875, -0.28466796875, 1.248046875, -0.84765625, 0.70947265625, 0.80322265625, -1.052734375, 0.115966796875, -0.98486328125, 0.267822265625, 0.26416015625, 0.953125, -0.290771484375, 1.5927734375, 0.328369140625, 1.671875, 1.09375, -0.3681640625, -0.496826171875, -0.2391357421875, 0.923828125, -0.15673828125, -0.5634765625, -1.1884765625, 1.77734375, -0.6005859375, 0.51513671875, 0.458984375, 0.01739501953125, -1.158203125, -0.61572265625, 1.8408203125, 0.1248779296875, -0.724609375, -1.4833984375, -0.333740234375, -0.68212890625, -0.59716796875, 0.443603515625, -0.1793212890625, -0.2568359375, 0.61572265625, 0.830078125, -1.0224609375, 0.07330322265625, -1.982421875, 0.03729248046875, 1.482421875, -0.02587890625, 0.68505859375, -0.814453125, 0.047943115234375, 0.77392578125, -1.0283203125, 0.595703125, -1.51953125, -0.5341796875, -1.4287109375, -1.6611328125, 0.55224609375, -0.82421875, -1.841796875, -1.3037109375, -0.31396484375, -1.4541015625, -0.306396484375, -0.366943359375, 2.162109375, 0.07305908203125, 0.58203125, -0.46142578125, 0.1861572265625, 0.06085205078125, 0.031951904296875, -0.485595703125, -1.5869140625, -1.6953125, -0.26318359375, 0.62646484375, -0.91650390625, -0.416015625, 1.0048828125, -1.939453125, -0.822265625, 1.2412109375, -0.422607421875, 0.335205078125, -0.098388671875, 1.0732421875, 0.50146484375, 0.12445068359375, 0.56298828125, 0.9072265625, -1.6025390625, 1.3876953125, -0.8779296875, 1.6669921875, -0.4453125, 1.2841796875, 0.57421875, 0.351318359375, -1.7548828125, -0.411865234375, 1.54296875, 0.90625, 0.7509765625, 1.2861328125, 0.007396697998046875, 0.05908203125, 0.62255859375, -0.8896484375, 0.161865234375, -0.4140625, 0.049407958984375, 0.873046875, 0.350341796875, -0.16845703125, 0.60205078125, 1.904296875, -1.068359375, 0.65869140625, -1.2939453125, -0.53466796875, 1.0849609375, 0.30224609375, -0.28466796875, -0.615234375, -1.0947265625, 0.79443359375, -1.5537109375, -0.546875, 0.61083984375, -0.9609375, -0.50634765625, -0.66015625, 0.7744140625, 0.5048828125, -1.453125, 0.99072265625, -0.767578125, -0.266845703125, 0.61083984375, -1.529296875, 0.3974609375, 0.433837890625, 0.71435546875, -1.126953125, 0.437744140625, -0.6083984375, -1.9990234375, 0.1142578125, -1.1376953125, 0.369140625, 1.9169921875, 0.52099609375, 0.6005859375, 0.38671875, -0.69384765625, -0.100830078125, -0.462890625, -0.449951171875, 0.4501953125, -1.3232421875, 0.7109375, -0.04376220703125, -0.57275390625, 0.73681640625, 1.6904296875, 0.35107421875, -0.59033203125, 0.2227783203125, 0.29833984375, 0.40966796875, 0.5, 0.9873046875, 0.626953125, -0.285400390625, 0.427734375, 1.654296875, 0.9716796875, -1.162109375, 0.57568359375, -0.41064453125, 0.01959228515625, -0.564453125, -0.93017578125, 0.76806640625, 0.39013671875, 0.7880859375, -0.66796875, 0.046600341796875, 0.9921875, 0.703125, -0.416748046875, -1.5927734375, -0.1260986328125, -0.60888671875, -1.140625, 0.5380859375, 0.08270263671875, 0.384033203125, -0.6396484375, 1.4384765625, -0.1229248046875, -0.9052734375, -0.79345703125, -0.623046875, 0.77099609375, 1.22265625, 1.3466796875, -0.043487548828125, -0.65673828125, 1.21484375, 0.280517578125, 1.4443359375, -0.74560546875, 0.50439453125, 0.56005859375, 0.04852294921875, -0.1395263671875, -0.08685302734375, 0.1868896484375, -0.64404296875, 0.7353515625, 1.134765625, -1.1259765625, 1.6708984375, 1.2822265625, -1.4921875, 0.364990234375, -1.5146484375, -0.394287109375, 1.5029296875, -0.4619140625, 0.5869140625, 0.293701171875, -0.2403564453125, -0.2342529296875, 1.3759765625, 0.14990234375, 1.51953125, -0.37744140625, -0.7353515625, 0.365478515625, -0.5849609375, 0.2401123046875, -0.59912109375, -0.86328125, -1.3603515625, 0.055023193359375, -0.141357421875, -1.1396484375, -1.3759765625, -0.7255859375, 0.3984375, -0.1112060546875, 0.9580078125, 0.421630859375, -0.734375, 0.320556640625, -0.07769775390625, 0.8369140625, 0.11871337890625, 0.32666015625, 0.77880859375, -0.2763671875, 0.3427734375, 0.78369140625, 0.529296875, -0.177978515625, 0.0848388671875, -0.53271484375, 0.99853515625, -0.3876953125, 0.6240234375, 0.312255859375, -1.8505859375, -0.09576416015625, -0.4931640625, 0.70166015625, -1.1142578125, 1.58984375, 1.66015625, -0.62109375, -0.77978515625, 0.403564453125, 1.6796875, -2.162109375, 0.351318359375, 1.498046875, 2.064453125, -1.1318359375, -1.6337890625, -0.5615234375, -0.06884765625, 1.0810546875, -0.7587890625, -0.0673828125, -0.57421875, -0.54541015625, -1.0947265625, 0.45849609375, 0.1512451171875, -0.611328125, -0.56982421875, 1.392578125, 0.420166015625, 0.14013671875, 0.89697265625, -1.1650390625, 0.607421875, 1.7314453125, -2.109375, 1.4580078125, -0.09521484375, -0.1905517578125, 1.01953125, -0.76904296875, 1.001953125, -0.290771484375, 0.73681640625, -1.326171875, 1.0400390625, -1.8330078125, 1.1572265625, -1.21484375, -0.46826171875, -1.67578125, -0.06866455078125, 0.0804443359375, -1.0078125, 0.08489990234375, 2.298828125, 0.72119140625, 0.0250091552734375, 1.0458984375, -1.13671875, 0.396728515625, 0.908203125, 0.431640625, 1.580078125, 0.490478515625, 1.7724609375, -0.509765625, 1.46875, 0.40576171875, 0.407958984375, -0.1510009765625, 0.326416015625, -1.0146484375, 0.701171875, -1.1259765625, -0.57666015625, -1.2587890625, -0.166015625, -0.124267578125, 0.128662109375, -0.9208984375, 1.59375, -1.1630859375, 0.72216796875, -0.38916015625, 0.5166015625, -2.76953125, -0.900390625, 0.830078125, 0.412109375, 2.283203125, 0.71923828125, 0.07342529296875, -1.75390625, 0.7001953125, -0.248779296875, 0.41650390625, 0.42626953125, -2.078125, 0.2275390625, 0.5478515625, -0.433349609375, -1.1240234375, 1.5673828125, 0.111328125, 0.457275390625, 0.42578125, 0.87939453125, -0.7255859375, -0.248046875, -0.65478515625, -0.48388671875, 1.23046875, 0.68994140625, -0.09332275390625, -1.6484375, -1.59375, 1.23046875, 0.039215087890625, -1.6806640625, 0.52490234375, 0.0736083984375, 0.2127685546875, 1.708984375, 2.05078125, -0.6689453125, -0.399658203125, -0.0858154296875, 2.9921875, 1.2587890625, -2.71875, 0.873046875, 0.246337890625, -0.2369384765625, -0.599609375, -0.125732421875, -2.05859375, -0.8974609375, -1.107421875, -0.2191162109375, -0.64990234375, 0.262939453125, 2.287109375, 1.6162109375, -1.9306640625, -0.1627197265625, 0.92236328125, -1.068359375, 0.8701171875, 0.953125, -0.51025390625, 0.56640625, -0.1663818359375, 1.6220703125, -2.212890625, 1.93359375, -0.9013671875, -0.306396484375, 1.3662109375, -0.62744140625, 0.5654296875, 0.0205230712890625, -3.029296875, 0.62939453125, 0.69580078125, -0.875, 0.89599609375, -0.385009765625, -0.435546875, 1.2626953125, -1.0205078125, -0.456298828125, -1.3935546875, -0.148681640625, -0.98095703125, -0.673828125, -0.31396484375, -1.5419921875, 0.85302734375, -0.53662109375, -0.1307373046875, 0.1546630859375, -0.71142578125, 0.57666015625, 0.55712890625, -0.72998046875, 1.43359375, -0.80712890625, 0.0034332275390625, 0.74951171875, -0.67822265625, -0.609375, -0.07647705078125, 0.98828125, 0.60986328125, 1.349609375, 0.26220703125, 0.82958984375, 0.144287109375, -1.42578125, 0.008148193359375, -1.6318359375, -0.62890625, 1.107421875, 1.783203125, 0.2010498046875, -0.347900390625, 0.0254669189453125, 0.47509765625, 2.13671875, 0.515625, 0.1663818359375, 0.087646484375, 1.0947265625, -0.87060546875, 1.7626953125, -0.8974609375, -0.75244140625, -0.78076171875, -0.1513671875, -0.36669921875, 0.0266876220703125, -0.84765625, -0.263916015625, -0.342041015625, -0.266845703125, -1.49609375, 0.82275390625, 1.8408203125, 0.169921875, 0.305908203125, -2.123046875, 2.376953125, 1.7333984375, -0.95068359375, -0.54150390625, -0.52294921875, -0.546875, -1.0390625, 0.08221435546875, -0.38720703125, -1.2919921875, 0.34375, -0.453369140625, 1.80078125, -1.3056640625, 0.4892578125, 1.91796875, 0.505859375, -1.462890625, 0.113525390625, 0.45751953125, 1.421875, -0.58203125, -0.01535797119140625, -1.291015625, -0.0004382133483886719, 0.8232421875, -0.0533447265625, 1.0087890625, 1.13671875, 1.232421875, -1.189453125, -1.7197265625, 0.62744140625, 0.6767578125, 1.49609375, -0.383056640625, 1.109375, -2.197265625, -1.18359375, -0.37939453125, 0.7109375, 0.316162109375, -1.490234375, 1.8203125, -1.3466796875, -0.388916015625, 1.0205078125, -0.25537109375, 1.20703125, 0.420166015625, -2.625, -0.76318359375, 1.2392578125, 0.333740234375, 1.5966796875, -1.0595703125, 0.80029296875, 0.2205810546875, -1.9384765625, 1.1533203125, 0.250244140625, 0.33984375, 1.1611328125, -0.280517578125, -0.84130859375, -1.328125, 0.734375, 1.962890625, 0.28759765625, 1.25, -0.185546875, 0.371826171875, 0.72119140625, 0.299072265625, 2.0703125, -0.340576171875, 0.49072265625, -0.258056640625, -0.8564453125, -2.005859375, 1.68359375, -0.98193359375, 0.0902099609375, 0.3896484375, -0.1739501953125, -0.326416015625, 0.95751953125, 1.751953125, 1.626953125, -1.609375, 2.08984375, 0.380859375, -1.50390625, 1.7265625, -1.0419921875, 0.355224609375, 0.395263671875, 0.354736328125, 0.44091796875, -0.74072265625, 0.91455078125, 0.7333984375, 0.183837890625, -0.62890625, -0.81982421875, 1.27734375, -0.94189453125, 1.029296875, -0.039764404296875, -0.36962890625, 1.0205078125, -0.69140625, 0.85498046875, 0.09552001953125, 0.35986328125, 0.326171875, 1.27734375, -0.6279296875, 1.9052734375, -1.203125, -1.7255859375, -0.6728515625, 1.7958984375, 1.0380859375, -1.5927734375, 0.296142578125, -0.7080078125, 1.2578125, -1.126953125, -1.29296875, -0.51611328125, -0.3642578125, -2.24609375, -1.0859375, -0.06292724609375, 0.7412109375, 2.15234375, 0.09619140625, -1.1650390625, 0.36279296875, -0.309326171875, 1.5322265625, -1.265625, -0.1766357421875, -0.3291015625, -0.57080078125, 0.53515625, 0.301025390625, -0.6640625, 0.375732421875, -1.0703125, -0.8466796875, 0.34326171875, 0.468994140625, -0.0197601318359375, -0.1495361328125, -0.271240234375, -0.66748046875, -0.32666015625, 0.37255859375, 1.2431640625, -1.4775390625, -0.54052734375, 1.037109375, -0.90234375, 0.8564453125, -0.416259765625, -0.4248046875, -0.74072265625, -1.1611328125, 0.1785888671875, 0.99853515625, -0.1346435546875, 1.0302734375, -0.38330078125, 0.65283203125, 1.873046875, 1.9189453125, 0.82958984375, 1.3095703125, -0.70458984375, 1.3408203125, -1.69921875, -0.1328125, -0.1746826171875, 1.119140625, -1.99609375, -1.7568359375, -0.3369140625, 0.29736328125, 1.609375, -0.748046875, 0.09423828125, -0.91650390625, 1.419921875, 0.21337890625, -1.0556640625, 0.57861328125, 1.62109375, -0.12261962890625, -0.74658203125, -0.2493896484375, -0.9208984375, -1.0576171875, 0.033050537109375, -1.349609375, -1.70703125, -0.7890625, 0.095947265625, -0.99267578125, 1.1240234375, -0.07940673828125, -0.70361328125, -1.2119140625, -0.6376953125, -0.07769775390625, -0.740234375, -1.005859375, -1.2783203125, 0.0660400390625, -1.455078125, -0.42431640625, 1.96484375, 0.7373046875, -0.04217529296875, 0.61962890625, -1.7265625, -1.1845703125, 0.7314453125, -1.7802734375, -0.873046875, 0.470703125, 1.107421875, 1.2744140625, 1.4912109375, -0.5048828125, 1.5517578125, -0.88818359375, 1.5625, 0.286376953125, 0.56982421875, -0.1300048828125, 1.5244140625, -2.2421875, -0.8193359375, -0.5576171875, 0.560546875, -0.81787109375, -1.240234375, -0.2357177734375, 1.01171875, -0.0560302734375, -0.398193359375, -0.400390625, -1.2080078125, -0.29443359375, -1.2138671875, -0.544921875, -1.78125, -0.84912109375, 2.373046875, -0.10394287109375, -0.79736328125, 0.587890625, 0.035369873046875, 0.85009765625, -1.6572265625, -0.48388671875, 0.06072998046875, 1.2099609375, -0.413818359375, -1.2880859375, 0.053955078125, 1.791015625, 1.587890625, -1.5087890625, -0.168212890625, -0.8203125, 2.904296875, 1.029296875, 0.1795654296875, -1.12890625, -0.09698486328125, -0.46875, 1.0263671875, 1.0478515625, 0.042999267578125, -0.0335693359375, 1.041015625, -0.810546875, 0.75048828125, 0.59619140625, 0.51318359375, -0.62109375, -0.509765625, 0.421142578125, -0.0408935546875, -0.0159759521484375, -1.2861328125, 0.92578125, 1.359375, -0.51171875, -0.951171875, 0.348388671875, -0.052032470703125, -1.3369140625, -1.5556640625, 0.2529296875, 1.38671875, 0.8818359375, -0.381103515625, 0.3193359375, 0.53173828125, -0.76708984375, 1.0546875, 0.8828125, -1.03125, 0.364501953125, -0.78076171875, -1.2666015625, -0.456787109375, 0.5087890625, -0.10595703125, -0.66748046875, 0.4482421875, -0.25146484375, 0.642578125, -0.194091796875, 0.4912109375, -0.1170654296875, -0.180908203125, -0.3818359375, -1.46875, 1.1796875, -0.33984375, 0.52294921875, -1.021484375, 0.038360595703125, -0.467041015625, -1.2724609375, -1.3818359375, -1.599609375, 0.83935546875, 0.3388671875, 0.666015625, -0.3828125, 1.4033203125, -2.05859375, -0.9267578125, 0.2215576171875, -0.26904296875, -0.36767578125, -1.4990234375, 1.884765625, -0.13330078125, -1.28125, -0.6728515625, -1.4072265625, 0.615234375, 0.105712890625, -0.517578125, 1.3837890625, 0.673828125, -0.6064453125, -1.185546875, 0.4892578125, -0.70068359375, 2.576171875, 0.68408203125, -1.306640625, 2.474609375, -0.56494140625, 1.5126953125, 1.240234375, 0.56494140625, 0.0008759498596191406, -0.3662109375, -0.050689697265625, 0.4775390625, -0.103271484375, -1.048828125, 1.283203125, 0.8984375, -0.8388671875, -0.3935546875, -0.9423828125, 0.1649169921875, 1.6064453125, -0.53173828125, -1.875, 0.6708984375, -0.223388671875, 0.7802734375, -1.1083984375, -0.051055908203125, 0.1478271484375, 0.042510986328125, -1.56640625, -0.48193359375, -1.6064453125, 1.458984375, 0.440185546875, 2.021484375, -0.1173095703125, 1.7607421875, -0.7158203125, 0.440673828125, 0.88037109375, -1.462890625, -0.2156982421875, -0.328125, -0.62451171875, -0.0362548828125, 0.45703125, 0.88232421875, -0.278564453125, 0.6474609375, 1.986328125, 1.09375, -0.42431640625, 1.5908203125, 1.3232421875, 0.309814453125, 0.7197265625, -0.6337890625, 0.97119140625, 1.212890625, -2.119140625, -2.15625, -0.37646484375, -0.3173828125, -1.5458984375, -0.80810546875, 0.463134765625, 1.19140625, -0.0465087890625, 0.008056640625, -0.97314453125, -1.431640625, -0.1092529296875, 1.056640625, 0.058349609375, 0.78759765625, -0.83203125, 0.5751953125, 0.498779296875, 0.77734375, -1.2138671875, -1.5625, -0.8203125, 0.11773681640625, -1.6826171875, 0.118408203125, -0.2025146484375, -0.1927490234375, -1.6953125, 0.270751953125, -0.91064453125, -1.6494140625, 0.63134765625, 0.052764892578125, 0.8896484375, 0.349853515625, -0.09381103515625, -0.818359375, 0.26220703125, 0.02783203125, -0.2401123046875, -0.9521484375, 2.3125, 0.5419921875, -1.1640625, -1.662109375, 0.061248779296875, 0.7958984375, -0.51904296875, 0.822265625, -0.29638671875, 1.70703125, -0.0237579345703125, -2.1953125, -0.32177734375, -0.10455322265625, -0.51123046875, 0.259765625, -0.0001837015151977539, 1.1845703125, 0.84716796875, -2.138671875, 0.33740234375, -0.66015625, 0.58740234375, 0.1551513671875, -1.8515625, -0.384765625, 1.0859375, -0.5517578125, 0.62158203125, 1.044921875, 1.50390625, -1.1640625, -0.22412109375, 1.13671875, 2.44921875, 0.443115234375, 0.78857421875, 1.2734375, 0.0251922607421875, 2.0234375, 0.418212890625, 0.9287109375, 0.23291015625, 1.33984375, 0.19970703125, 1.7939453125, 1.19140625, -0.367919921875, 2.171875, 0.12469482421875, 0.176025390625, -1.5732421875, -2.130859375, 0.791015625, -0.0186767578125, 0.472412109375, 1.73828125, 0.08111572265625, -0.0811767578125, -0.348388671875, 1.216796875, -0.341796875, 0.12066650390625, -1.0498046875, -0.1739501953125, -0.82958984375, 1.0673828125, -1.359375, 0.4638671875, -0.619140625, -1.4453125, 0.2919921875, -0.8623046875, -0.27294921875, 0.646484375, 1.65625, -0.4560546875, -1.119140625, 1.3271484375, -1.642578125, 0.7646484375, -1.1103515625, 2.1015625, 0.82568359375, -2.0625, -0.875, 0.365478515625, -0.385009765625, -0.009124755859375, -0.51904296875, 0.257568359375, -0.38818359375, 0.69970703125, -0.3994140625, 1.1953125, 0.142333984375, -0.61474609375, 1.7041015625, 0.50732421875, 0.71240234375, 0.7548828125, 1.03515625, -0.1888427734375, 1.6796875, -0.919921875, 1.0146484375, -0.62353515625, -0.93359375, -0.1737060546875, 0.177978515625, 0.98486328125, -0.8642578125, -1.734375, 1.7578125, -1.2119140625, -0.03204345703125, 0.580078125, -0.1236572265625, 1.068359375, 1.0634765625, 0.496826171875, 0.81787109375, 1.732421875, -2.52734375, -0.67333984375, 0.53271484375, -0.2333984375, 1.3583984375, 0.2509765625, 1.05859375, -0.76171875, 1.3818359375, 0.90234375, 0.11077880859375, 0.37255859375, -1.0751953125, -2.421875, 0.0870361328125, -0.04022216796875, 1.478515625, 0.70361328125, 0.4638671875, 0.271728515625, -0.10260009765625, -0.52685546875, -1.3134765625, 1.96875, 1.263671875, -0.54736328125, -1.34765625, 0.796875, -1.6416015625, 0.445556640625, 1.7470703125, 0.3486328125, -1.94921875, 1.28515625, 0.032562255859375, 0.88037109375, 0.18017578125, -1.740234375, 1.0615234375, 0.22314453125, -0.65771484375, -0.701171875, -0.75, 1.1591796875, 0.3515625, -0.38037109375, -0.77490234375, 0.443603515625, -0.619140625, 0.66796875, 0.90673828125, -0.484375, -0.2080078125, 0.2283935546875, 0.49365234375, -0.03216552734375, 0.232666015625, 1.814453125, 0.50244140625, -0.970703125, -1.0380859375, 0.8115234375, -0.8154296875, 0.70849609375, -1.267578125, -0.392578125, 1.537109375, 0.1334228515625, 0.1680908203125, -1.14453125, 0.5732421875, 0.6162109375, -1.65625, -0.33740234375, 0.04644775390625, 0.378173828125, 0.00537872314453125, -0.51416015625, -0.45947265625, 0.9658203125, -0.11932373046875, -0.1280517578125, 0.1961669921875, -0.492919921875, 1.05859375, 0.035247802734375, 1.2294921875, -0.9580078125, 0.80517578125, -0.57470703125, 0.66455078125, -0.83837890625, 0.27099609375, 0.67138671875, -0.1583251953125, -0.230712890625, 0.0498046875, 0.6064453125, 0.69189453125, 0.46630859375, -0.09747314453125, 0.03155517578125, 0.2109375, 1.134765625, -0.01019287109375, 1.2119140625, 0.466796875, -0.90478515625, -1.7119140625, 0.095703125, 1.599609375, -0.60009765625, -0.77587890625, 0.54150390625, 0.409912109375, -1.3935546875, -0.1339111328125, -0.77490234375, 0.0047149658203125, -0.51025390625, -0.23046875, -0.8232421875, 0.1192626953125, 0.294189453125, -0.058990478515625, 0.7294921875, -0.07373046875, -0.68310546875, -2.19140625, -1.1904296875, -0.54052734375, 2.126953125, -0.66650390625, -0.47216796875, -0.4453125, -0.7861328125, 1.6376953125, -1.0634765625, -1.271484375, 0.67333984375, -0.52001953125, -0.962890625, -0.8828125, 0.442626953125, -1.47265625, 0.865234375, 0.548828125, -0.69921875, -0.70849609375, 1.8330078125, 0.2265625, 0.68212890625, 0.33447265625, 0.22119140625, 0.08648681640625, -0.671875, 0.7099609375, 1.0107421875, -0.443603515625, 0.1856689453125, -0.1138916015625, -0.1461181640625, 0.740234375, 1.2880859375, -0.5927734375, -1.390625, -1.6484375, -0.60498046875, 0.2509765625, 0.405029296875, -0.361572265625, -0.92529296875, -0.33984375, -0.92041015625, 0.03704833984375, -0.440185546875, -0.49560546875, -2.005859375, 0.09393310546875, 1.224609375, -0.6396484375, -0.371337890625, 0.671875, 0.85400390625, 1.2783203125, 0.869140625, 1.18359375, 0.927734375, 1.1171875, -0.43798828125, 0.019561767578125, -0.447509765625, -0.4541015625, -0.73876953125, 0.240478515625, 0.2093505859375, 0.69921875, -0.39453125, -0.2373046875, 0.970703125, 0.615234375, -0.56494140625, -0.125, -1.541015625, 1.0439453125, -0.08160400390625, -0.099365234375, -2.001953125, -1.0888671875, -0.77490234375, 0.330322265625, -1.654296875, 1.6474609375, -0.78515625, 0.1304931640625, -0.08282470703125, 0.10833740234375, 0.272705078125, 2.666015625, -1.40234375, -1.6171875, 0.650390625, -0.712890625, 0.5498046875, 1.0517578125, 2.0625, -0.28125, -0.0751953125, 0.9833984375, 0.2042236328125, -1.181640625, 1.072265625, -1.7353515625, -1.1728515625, -0.329345703125, -0.442626953125, -0.1090087890625, -0.06707763671875, -0.11456298828125, 1.7724609375, 0.271728515625, 0.0283966064453125, 1.072265625, -0.63330078125, -0.1815185546875, 1.064453125, 1.5087890625, -0.375, 1.0166015625, 2.03515625, 0.9091796875, -1.5205078125, 1.025390625, 0.88427734375, -1.01953125, 0.0247039794921875, -0.98291015625, 0.6640625, -0.46826171875, 1.0595703125, -1.4013671875, -0.1121826171875, 1.67578125, 2.322265625, 0.8486328125, -0.07525634765625, 1.263671875, 0.0897216796875, 1.0908203125, -0.8681640625, -0.1571044921875, -1.6259765625, -0.060821533203125, 1.673828125, 1.23828125, -0.60595703125, 0.9091796875, -1.373046875, -1.25390625, 0.607421875, 0.7216796875, -0.51171875, -0.461669921875, 0.30859375, -0.69384765625, 0.89111328125, 0.68896484375, 0.65966796875, -1.7841796875, -0.55224609375, 0.259521484375, 0.29541015625, 0.2425537109375, -0.474365234375, 1.451171875, -0.1480712890625, 0.023284912109375, 0.09686279296875, 0.74853515625, 0.8193359375, 1.826171875, 0.80810546875, 0.11767578125, -1.6884765625, -0.2476806640625, -0.9375, -1.060546875, -1.9287109375, 1.92578125, -1.03125, 0.43359375, 0.62158203125, 0.6484375, -0.0628662109375, -0.5234375, 0.158447265625, -0.62890625, -1.0390625, -1.8115234375, -0.355712890625, 0.78271484375, -0.6943359375, 0.9345703125, -1.1376953125, 0.49072265625, -0.76611328125, 1.0244140625, 0.26171875, 0.87841796875, -0.2037353515625, -1.6298828125, -1.4072265625, 0.326904296875, -0.409912109375, 0.26806640625, -2.3359375, 1.9775390625, 0.56640625, 0.1407470703125, 1.359375, -0.396240234375, -0.57275390625, -1.09765625, -1.46484375, -1.1201171875, 0.615234375, 0.1070556640625, -1.1640625, 0.27783203125, -0.56103515625, -0.931640625, -0.31396484375, -0.74072265625, -0.75732421875, -0.96630859375, 0.0154266357421875, -0.343017578125, -0.9267578125, -0.2406005859375, -1.48046875, -0.802734375, 0.325439453125, 2.05078125, 0.9462890625, -0.75927734375, 0.4013671875, 2.30859375, 0.84375, 1.384765625, 0.271728515625, -0.11151123046875, -0.63623046875, 0.0955810546875, -0.6904296875, -1.568359375, 0.1973876953125, -1.30859375, 0.302734375, -0.86376953125, -0.76220703125, -0.67724609375, 0.317626953125, -1.9228515625, -0.305908203125, 0.39404296875, -0.6943359375, 0.6943359375, 0.4736328125, -1.0966796875, 0.483642578125, -2.072265625, -0.1734619140625, -2.62890625, 0.61181640625, 0.60546875, -0.57470703125, -0.90380859375, 1.2734375, -0.32666015625, -0.15576171875, 0.3212890625, -0.110107421875, 0.02142333984375, -0.42578125, 0.1990966796875, 0.1954345703125, -0.449462890625, 0.931640625, 0.0440673828125, 0.56103515625, -0.93994140625, 1.4150390625, -0.77099609375, 0.8837890625, -1.4638671875, -0.55078125, 0.432373046875, -0.732421875, 0.052032470703125, -1.58203125, -1.2998046875, 1.474609375, -0.025543212890625, -0.828125, -0.52734375, 0.791015625, 0.127685546875, -0.0377197265625, -0.27587890625, 0.3525390625, -0.3310546875, -1.162109375, 1.21875, 1.0517578125, 0.8525390625, -0.254150390625, -0.187255859375, -1.015625, 2.4140625, -1.3330078125, -0.3466796875, -0.361083984375, 3.365234375, -1.65234375, -0.275146484375, 1.0244140625, 0.432861328125, -1.4912109375, -0.537109375, -0.98193359375, -0.86669921875, -1.6904296875, 2.7890625, 1.0244140625, -0.15673828125, 0.34423828125, -1.0087890625, 0.085205078125, 0.3359375, 0.9736328125, -1.169921875, 1.20703125, 0.2427978515625, 0.186279296875, 0.6923828125, 0.07342529296875, -0.022247314453125, 0.352294921875, -1.0927734375, -0.048583984375, -0.34228515625, 0.81982421875, -0.030792236328125, -1.193359375, -0.88623046875, 0.6337890625, -0.316162109375, 1.32421875, -1.1494140625, 0.939453125, 0.74658203125, 1.3193359375, -0.7294921875, -0.69677734375, 1.5908203125, 0.62353515625, 0.08709716796875, -0.53369140625, -0.38134765625, -2.6875, 0.478759765625, 0.4013671875, -0.59716796875, 0.69091796875, -0.62548828125, -0.337646484375, -0.65478515625, 0.53662109375, 2.42578125, -0.051788330078125, -1.54296875, 0.919921875, -1.158203125, -1.2158203125, 0.638671875, 0.7646484375, 0.94677734375, -1.12890625, -0.8125, -0.0540771484375, -0.265380859375, 1.2998046875, 0.68359375, -0.07891845703125, 0.349853515625, 0.6005859375, -0.8828125, -0.284912109375, -0.71826171875, 0.93017578125, -0.63818359375, -1.232421875, -0.26318359375, -0.84228515625, -1.20703125, -0.3212890625, 1.9365234375, -2.330078125, -0.81201171875, 0.2181396484375, 0.114013671875, -0.43798828125, -0.7001953125, 0.447021484375, 0.6806640625, 0.044769287109375, 0.8271484375, 0.60498046875, -0.70703125, -1.833984375, -0.5263671875, -0.6787109375, -1.9462890625, 1.115234375, 0.60986328125, 0.353759765625, -1.2431640625, 0.19482421875, -1.9619140625, 0.23828125, -1.0244140625, 1.1357421875, -0.7900390625, -0.5556640625, 1.2900390625, -2.26953125, 1.2724609375, -1.1298828125, -0.310302734375, 0.393798828125, 2.1015625, 1.24609375, 0.09918212890625, 0.128173828125, 0.8271484375, -0.186767578125, -1.4248046875, 0.0377197265625, -1.5185546875, 1.0126953125, -1.685546875, 2.052734375, 0.71240234375, -0.4287109375, 0.38427734375, 0.978515625, -2.35546875, 1.4052734375, -0.448974609375, 0.85009765625, -0.48291015625, -2.310546875, 2.673828125, -0.59228515625, -0.0643310546875, -0.0289306640625, 0.11785888671875, -1.296875, -1.525390625, 1.6025390625, 0.74609375, -0.6669921875, -1.5771484375, 0.2041015625, -0.0728759765625, -0.98974609375, -2.072265625, -0.30419921875, 0.4453125, 2.056640625, 0.425537109375, -0.315673828125, 0.8720703125, 1.1318359375, 1.44921875, 1.0419921875, -0.391845703125, -0.146484375, 1.5263671875, 0.62060546875, -0.246337890625, -0.06201171875, 1.0, -0.016632080078125, 0.5556640625, -1.796875, -1.001953125, -0.394287109375, -1.5458984375, -0.9736328125, -0.21728515625, -1.3720703125, 0.1922607421875, 0.2430419921875, -0.148193359375, -0.79052734375, -0.8017578125, 0.71484375, 1.126953125, 0.546875, 1.763671875, -0.335205078125, 1.1787109375, 0.5771484375, -0.2293701171875, -0.387939453125, -0.9814453125, 0.4150390625, 0.84326171875, -0.211181640625, 0.02301025390625, 0.156005859375, 0.9931640625, -0.4267578125, 0.218505859375, 0.0026912689208984375, -0.61328125, -0.364990234375, -0.25048828125, -1.416015625, 1.1025390625, 0.56640625, -0.5361328125, 1.0888671875, -0.81396484375, 0.358154296875, -0.07037353515625, -2.341796875, 0.875, 1.5859375, 0.318603515625, 1.966796875, 0.424560546875, 0.220947265625, -0.09002685546875, 0.2410888671875, 0.6884765625, -0.53125, -0.200927734375, -2.556640625, -0.059661865234375, -0.4580078125, -0.0467529296875, 0.36767578125, 0.54931640625, 0.472412109375, -0.0027675628662109375, -1.05859375, -1.220703125, 0.2005615234375, -1.0732421875, 0.113037109375, -0.55615234375, 0.041473388671875, 0.77685546875, 0.0237579345703125, -1.08984375, -0.72705078125, -1.3818359375, -0.06597900390625, -0.84619140625, -1.4306640625, -0.6494140625, 1.505859375, 2.068359375, 0.50927734375, 0.50927734375, -0.9990234375, -1.185546875, 0.2274169921875, 1.4794921875, 1.9267578125, -0.115966796875, -0.2127685546875, -0.2734375, 0.8466796875, 0.81494140625, -0.447265625, 0.30029296875, 0.2310791015625, -2.802734375, -0.2442626953125, -0.497802734375, 1.2958984375, -0.478515625, 0.11236572265625, -1.1865234375, -0.1431884765625, -1.1884765625, 1.3955078125, -0.462890625, -1.3134765625, 1.1376953125, 1.404296875, 0.20751953125, 1.05078125, 0.65576171875, -0.166748046875, 0.465087890625, 1.0166015625, -0.6953125, 0.8544921875, 0.64990234375, 0.39306640625, -0.63134765625, 0.82666015625, 0.387939453125, -0.5439453125, 1.91015625, 1.25390625, 1.2138671875, 0.0633544921875, 1.4833984375, -0.370361328125, 0.287109375, 0.384521484375, -0.9931640625, 0.67529296875, 0.63134765625, 0.259765625, 0.250244140625, 0.50830078125, 0.498046875, -0.4697265625, 1.2451171875, -1.3857421875, -1.146484375, 0.52734375, 0.246826171875, -0.248779296875, 0.86572265625, -0.77783203125, 1.419921875, 0.626953125, -0.88330078125, -0.1409912109375, -0.06201171875, 0.9267578125, 0.2342529296875, 0.71875, 0.4326171875, -0.59033203125, -0.67626953125, -0.61572265625, -0.1485595703125, 0.70263671875, 0.256103515625, 2.2265625, 0.50537109375, 0.6572265625, 1.3427734375, -0.367431640625, 0.6513671875, -0.6669921875, -0.6142578125, 0.220947265625, -0.364013671875, -0.383056640625, 0.0118865966796875, -1.564453125, -0.55712890625, -0.6806640625, 1.7099609375, -0.375244140625, -0.767578125, 0.476318359375, 0.58984375, 1.8935546875, -0.56396484375, 1.3544921875, -0.11944580078125, -1.3916015625, -1.7158203125, -2.11328125, 0.057769775390625, 0.08319091796875, -1.701171875, 0.2047119140625, 0.67529296875, 0.03204345703125, -0.978515625, -0.78125, -0.440673828125, 0.6845703125, 0.8896484375, -1.1875, 0.931640625, -0.8857421875, -0.08349609375, 1.4150390625, -1.5634765625, -1.505859375, -0.94921875, 0.7099609375, -1.078125, 0.06719970703125, 1.650390625, -1.712890625, -0.208740234375, 1.92578125, -0.400634765625, -1.21875, -0.06842041015625, 0.38232421875, 0.001979827880859375, -0.6533203125, -0.75927734375, -2.57421875, 3.208984375, -1.2158203125, 0.0989990234375, -0.11004638671875, 2.423828125, 1.751953125, -1.138671875, 0.21044921875, 0.013641357421875, -1.0927734375, -1.0478515625, 0.716796875, 1.099609375, 1.8349609375, 1.3095703125, -0.9716796875, -1.0146484375, 0.8125, -0.034149169921875, 0.65869140625, 1.16015625, -0.4189453125, 1.1318359375, 0.6572265625, -0.72314453125, 0.1485595703125, 0.41162109375, 0.4541015625, 1.1357421875, 1.845703125, 0.85546875, -1.3251953125, -0.57470703125, 2.140625, -0.364990234375, 1.15234375, -0.6767578125, -1.369140625, -0.0064697265625, 2.783203125, -1.42578125, 2.20703125, 0.1815185546875, 0.412841796875, -0.4970703125, 0.60400390625, 0.1527099609375, 0.59521484375, 0.0435791015625, -1.109375, -1.453125, -0.314208984375, -0.68603515625, -0.002574920654296875, -1.2861328125, -0.1617431640625, -0.11395263671875, -1.453125, 0.491455078125, -0.6259765625, 2.005859375, 0.411865234375, -0.810546875, 1.4658203125, 0.681640625, -1.2265625, -0.432373046875, -0.55029296875, -0.556640625, 0.356689453125, 1.1455078125, 1.1591796875, 1.2470703125, 0.89208984375, 0.57080078125, 0.79296875, 0.4794921875, 0.10064697265625, 0.90478515625, -1.361328125, 0.662109375, -0.014129638671875, 1.97265625, -0.036773681640625, 1.1923828125, -0.85107421875, -1.015625, 2.078125, 2.333984375, 0.787109375, -0.69921875, 1.55859375, -0.0113677978515625, 0.0340576171875, -0.109619140625, 1.849609375, 0.75537109375, 1.5810546875, -2.078125, 0.1334228515625, -0.299072265625, 0.26953125, 0.99658203125, 0.64892578125, -1.6884765625, 0.9443359375, 2.21875, -0.50048828125, -1.3369140625, -0.55517578125, 1.3115234375, -2.021484375, 1.6669921875, -0.91455078125, 0.5029296875, 0.1661376953125, -1.025390625, 0.62646484375, 0.0188140869140625, 2.95703125, -0.619140625, -0.43603515625, -0.70751953125, 0.81005859375, 1.078125, -1.8076171875, 0.296142578125, 0.52734375, 1.5185546875, -1.99609375, -0.44287109375, -0.11956787109375, -1.2939453125, -1.6982421875, -0.82275390625, -0.962890625, -2.58984375, 1.1064453125, 0.982421875, -0.59326171875, -0.355712890625, 1.291015625, 0.25927734375, 0.032562255859375, -1.5322265625, 0.52392578125, -1.5546875, 0.59765625, -0.367431640625, 0.6767578125, 0.425048828125, 0.11358642578125, 0.302490234375, -0.89013671875, 0.2364501953125, 0.44189453125, 1.2890625, -1.109375, -0.214111328125, -1.2998046875, -0.5478515625, -0.08197021484375, 0.64208984375, 0.96435546875, 1.5556640625, 0.6572265625, 1.498046875, -0.452392578125, -0.7294921875, 0.1873779296875, -0.27392578125, -0.77197265625, -0.849609375, -0.7578125, -0.30859375, 0.7236328125, 0.7548828125, -0.12451171875, -0.2496337890625, -0.56640625, 1.2421875, -1.189453125, -1.5966796875, -2.03125, -0.07525634765625, 1.3408203125, 1.046875, -1.0673828125, -0.5126953125, -0.92333984375, -0.1019287109375, -0.8193359375, 0.919921875, -0.2178955078125, -0.2095947265625, 0.1412353515625, -0.77197265625, 0.30419921875, -0.098876953125, -0.443603515625, 0.529296875, 2.421875, 0.306396484375, 0.1905517578125, 1.8896484375, -0.0848388671875, 1.40625, 0.47314453125, 0.6259765625, 1.4169921875, -0.93359375, 1.033203125, 0.059112548828125, -0.2279052734375, -1.517578125, 0.2342529296875, 0.39794921875, -0.8193359375, -1.376953125, -1.0263671875, 0.45068359375, 1.5703125, -0.9716796875, -1.3359375, 1.8818359375, -1.0703125, 1.0146484375, -0.81591796875, 0.7353515625, -2.7109375, -0.1375732421875, 1.3115234375, -1.390625, -1.28125, -0.59423828125, -0.98193359375, 0.22900390625, 1.701171875, 0.0809326171875, -1.5224609375, -0.15966796875, 0.705078125, 1.5625, -1.5361328125, -0.15771484375, 0.8349609375, -0.194580078125, -1.658203125, -1.1728515625, -1.513671875, -0.257568359375, -1.4296875, -0.55615234375, 0.703125, -0.254150390625, 0.72119140625, -0.76171875, 1.7138671875, -0.548828125, -0.368408203125, 1.34765625, -0.1793212890625, 0.386962890625, 0.47412109375, 1.119140625, 1.2353515625, 0.80126953125, -1.8369140625, -0.82275390625, -0.583984375, 0.8984375, -0.287841796875, 0.1829833984375, -0.755859375, 0.30908203125, -0.94970703125, -0.86181640625, -1.7216796875, -0.61962890625, 0.10736083984375, 1.560546875, 0.11395263671875, 1.1171875, 0.058319091796875, 0.5771484375, 0.60498046875, 1.3037109375, -0.92333984375, 0.2529296875, -0.0955810546875, -0.19384765625, -0.00341796875, -0.4189453125, 0.1064453125, 1.51171875, -1.6904296875, -0.64501953125, -1.435546875, -0.397705078125, -0.097900390625, -1.5654296875, 2.73046875, 0.7451171875, 0.27587890625, -1.1953125, 0.91455078125, 0.87451171875, 0.88671875, -1.125, 0.6630859375, 0.427734375, -1.5341796875, 0.849609375, -1.7451171875, 0.1103515625, -1.4697265625, -0.1781005859375, 0.80615234375, 0.84814453125, 0.75, 1.1171875, 2.2421875, 0.59033203125, 0.251220703125, -0.81787109375, -0.74951171875, 0.962890625, -0.5048828125, -0.417236328125, 0.35009765625, -0.5458984375, 0.111083984375, -0.64697265625, 1.28515625, -0.053192138671875, 0.158447265625, -1.2099609375, 1.2841796875, 0.17333984375, -0.8603515625, 0.35302734375, 0.420166015625, -0.69384765625, -0.63916015625, -0.324462890625, 0.07037353515625, 0.5029296875, 1.72265625, 0.53857421875, 0.53857421875, -1.4755859375, -0.40234375, -0.1317138671875, -0.019439697265625, -1.2841796875, -0.9931640625, -2.46875, -1.3330078125, -0.720703125, 0.580078125, -0.294921875, 1.6220703125, -1.01171875, -0.19091796875, 0.63623046875, 0.18115234375, -1.724609375, -0.95166015625, -1.349609375, -1.1611328125, -0.475830078125, 0.6513671875, 0.25390625, -0.09326171875, -1.6826171875, 0.72412109375, -0.49072265625, -0.142578125, 0.24609375, 0.45751953125, 0.53369140625, -1.75390625, -0.58544921875, -1.603515625, 0.2841796875, 1.9453125, 0.63818359375, -0.6962890625, -0.55322265625, -0.66259765625, 1.0283203125, 0.515625, -0.353515625, -0.08245849609375, 1.431640625, -0.47119140625, 0.314453125, -0.50830078125, -0.923828125, -1.517578125, 0.2373046875, 1.6025390625, -0.572265625, 0.191162109375, 0.2171630859375, 0.060333251953125, -0.194091796875, 0.293212890625, -1.365234375, 0.76220703125, -0.1759033203125, -0.2479248046875, 0.37451171875, 0.0482177734375, 0.5439453125, -0.35302734375, -0.57470703125, 0.9365234375, 2.015625, 0.338623046875, 0.61279296875, 1.201171875, -2.0859375, 0.673828125, 0.2030029296875, 1.4443359375, -1.4638671875, 1.111328125, -0.09771728515625, -1.572265625, -0.76904296875, 1.7685546875, -0.970703125, 1.193359375, -0.80859375, -0.61181640625, -0.052825927734375, 0.42822265625, 0.05517578125, 0.397705078125, -1.0615234375, -0.01995849609375, 0.110595703125, -0.27099609375, 1.8154296875, 1.4462890625, -1.896484375, 0.357177734375, 0.96484375, -1.78515625, 1.4677734375, -0.4482421875, 0.125732421875, -1.3662109375, -0.7216796875, -0.306640625, -1.09375, -0.283203125, -1.2001953125, -0.95556640625, 0.08038330078125, -0.459228515625, 1.359375, 0.214111328125, -2.5703125, 0.34228515625, -1.4912109375, -0.286865234375, -0.791015625, -1.634765625, 0.215576171875, 1.3271484375, -1.232421875, -1.271484375, -0.54541015625, -0.1082763671875, 1.537109375, -0.46630859375, -0.0728759765625, -1.318359375, 0.0677490234375, -0.3427734375, 0.349609375, -1.189453125, -0.60693359375, 1.6240234375, -0.418212890625, -0.04730224609375, -0.30859375, -0.396240234375, -0.6689453125, 0.56396484375, 0.042755126953125, 0.8056640625, -0.0012340545654296875, 0.72998046875, 2.072265625, -1.2685546875, -0.4609375, 1.2021484375, -0.397705078125, -1.5029296875, 0.65283203125, -0.982421875, 0.89794921875, -0.76953125, -1.033203125, -1.021484375, -1.3759765625, 0.442138671875, -1.1611328125, 1.267578125, 0.078857421875, 1.0205078125, -0.12396240234375, -1.037109375, -2.828125, 1.6416015625, 0.48681640625, 0.1121826171875, 1.76171875, -0.0556640625, -0.5458984375, 0.7890625, 0.0992431640625, 0.50537109375, -1.021484375, -0.8984375, -0.1224365234375, 0.48876953125, 0.342529296875, -0.66796875, 0.1392822265625, -0.402587890625, 2.03515625, 1.7412109375, 0.806640625, -0.177001953125, -0.253173828125, 0.3212890625, 0.927734375, -0.493896484375, -0.2490234375, -0.397216796875, -0.3408203125, -0.271728515625, 0.37255859375, -1.8291015625, 0.84130859375, 0.301513671875, -0.59619140625, 0.4306640625, 1.2490234375, 0.98779296875, 0.3935546875, 0.088134765625, 1.0908203125, -0.29736328125, -0.59521484375, -1.6572265625, 1.439453125, 1.0908203125, 0.83544921875, 0.34716796875, -0.6259765625, -1.4609375, -2.328125, -0.76123046875, -1.5185546875, 0.53173828125, -0.40673828125, -0.671875, 2.0859375, 0.5322265625, 0.301025390625, -0.199951171875, -0.42578125, 2.013671875, 0.62060546875, 0.23974609375, -0.06298828125, -0.39306640625, 0.759765625, 0.5810546875, -3.083984375, 1.3798828125, -0.47314453125, -0.463623046875, 0.52392578125, 0.56689453125, 0.9892578125, 1.9033203125, 2.4375, -0.56787109375, 0.57470703125, 0.88623046875, 0.1468505859375, 0.006664276123046875, -0.1788330078125, 2.625, 0.18701171875, 0.056610107421875, 0.91796875, 0.6357421875, 0.794921875, -0.312744140625, -1.625, 0.9541015625, -0.40771484375, -1.513671875, -0.468017578125, 1.6806640625, -0.31201171875, 1.345703125, 1.4208984375, 1.3623046875, -0.8642578125, -0.18994140625, -0.01276397705078125, 0.33349609375, -0.3056640625, -1.0625, -0.9833984375, 0.455322265625, 2.130859375, -1.30078125, -0.273681640625, 1.935546875, 0.07757568359375, -0.09613037109375, 1.0439453125, 0.5029296875, -1.6982421875, -0.623046875, -0.8955078125, 0.49560546875, 1.2392578125, -0.1343994140625, 0.42822265625, 1.642578125, -0.8037109375, 0.97802734375, -1.3974609375, 0.51513671875, -0.2208251953125, -1.248046875, 0.974609375, 0.50146484375, 0.509765625, 0.1527099609375, 1.5322265625, 0.0311737060546875, 0.328857421875, -0.2276611328125, 0.03558349609375, 1.1474609375, -0.419677734375, -0.10540771484375, 0.87255859375, 0.09783935546875, 0.3369140625, 2.267578125, -1.962890625, -1.517578125, -0.66748046875, -1.001953125, 0.348876953125, 0.55419921875, 0.46484375, 0.5859375, -0.79345703125, 0.69921875, -0.68115234375, -1.068359375, 1.0771484375, -0.6748046875, 1.9306640625, -0.55517578125, -1.7080078125, -0.87451171875, 0.157470703125, 0.296142578125, -0.0001685619354248047, -0.076171875, -1.021484375, -2.26953125, 0.223388671875, -0.7529296875, -1.9248046875, -0.38232421875, -1.916015625, -1.23046875, -1.4130859375, -0.0172119140625, -0.7001953125, -0.0193634033203125, 0.17333984375, 0.1290283203125, 0.6787109375, 0.82763671875, -0.59130859375, -0.87548828125, 0.53662109375, 1.158203125, -0.923828125, 1.7177734375, -0.71728515625, 0.05731201171875, 0.459716796875, -1.4599609375, -0.39013671875, -0.2115478515625, 0.0087432861328125, 0.185791015625, -0.08111572265625, 0.146484375, -0.67138671875, -1.109375, -0.2315673828125, 0.94580078125, 0.43994140625, -0.09954833984375, -0.388671875, -1.421875, -1.8759765625, 0.0223541259765625, 0.54443359375, -0.73681640625, 1.1435546875, 0.6484375, -0.63427734375, 1.1904296875, -1.314453125, 0.1954345703125, 1.701171875, 0.90673828125, 1.0869140625, 1.48828125, 0.82470703125, -0.8671875, -0.5576171875, -0.3037109375, 0.73876953125, -0.062286376953125, -0.57470703125, 0.036895751953125, 0.7294921875, -1.7421875, 0.1524658203125, 0.245849609375, 0.2958984375, 1.728515625, 0.348876953125, -0.41064453125, -0.62451171875, -0.50927734375, 0.70849609375, 0.68798828125, -1.5478515625, -0.9248046875, -1.865234375, -0.284912109375, -0.278564453125, -0.86767578125, -1.1259765625, -0.83837890625, 0.12347412109375, -0.341064453125, -2.51953125, -2.341796875, 0.06817626953125, -0.29345703125, 1.142578125, 0.96630859375, -0.4716796875, 1.833984375, 2.11328125, 0.0127716064453125, 0.87255859375, 0.54541015625, -0.277587890625, 0.1597900390625, 0.122802734375, 0.599609375, 1.19921875, -1.412109375, 0.82275390625, -0.390869140625, -0.100341796875, -0.8212890625, -0.07305908203125, 2.71484375, -0.501953125, 0.68505859375, 1.4013671875, 0.35546875, -0.63720703125, 0.2193603515625, -1.0595703125, -1.1123046875, 1.6630859375, 0.0205078125, -1.7197265625, 1.7666015625, -1.1298828125, 0.06854248046875, -0.278564453125, -0.1300048828125, -0.9599609375, -1.791015625, 0.849609375, -1.1123046875, -0.580078125, -0.8544921875, -1.6357421875, -0.54052734375, -0.419921875, -0.007770538330078125, 0.73828125, -0.62451171875, -0.658203125, 1.3310546875, 1.1669921875, 0.37841796875, 0.08233642578125, 0.30322265625, -0.5234375, 1.294921875, 1.0263671875, -0.53125, -0.63720703125, 0.05792236328125, -1.732421875, -0.141357421875, -1.16015625, -0.005767822265625, 0.07806396484375, -0.385498046875, -0.005008697509765625, 1.1435546875, -0.11480712890625, 1.0732421875, 1.126953125, 0.58203125, -0.3115234375, 1.15234375, 0.33642578125, -0.42041015625, 0.8623046875, -0.347412109375, 0.83251953125, 0.47705078125, -0.525390625, -0.1026611328125, -0.08551025390625, 1.521484375, -1.623046875, -1.138671875, -0.287841796875, -0.876953125, 0.47998046875, -1.01171875, -0.042510986328125, 0.044952392578125, -0.12249755859375, -1.22265625, -0.5859375, -1.7861328125, 0.43603515625, 0.517578125, 0.85498046875, -1.123046875, -0.415283203125, 0.0771484375, 0.53369140625, -1.1884765625, 0.390625, 0.70263671875, 0.7646484375, -0.08935546875, -1.0771484375, 0.388671875, 0.1697998046875, -0.8720703125, 0.1746826171875, 2.2265625, 1.0859375, -1.634765625, 0.1229248046875, -0.411376953125, 0.72265625, -0.2587890625, 0.84228515625, -0.328369140625, 1.109375, 0.7236328125, 1.208984375, 0.1383056640625, 0.189453125, -0.8740234375, 0.2088623046875, 0.98779296875, -0.05865478515625, 0.85888671875, 1.4794921875, 2.201171875, -0.322509765625, 1.6337890625, 0.1939697265625, 0.6689453125, 1.1748046875, 0.44189453125, 0.2568359375, 0.2108154296875, -0.381103515625, -0.270263671875, 0.533203125, 0.45947265625, 0.2147216796875, 0.213134765625, -0.60888671875, -0.250244140625, 1.4765625, 1.822265625, -0.6142578125, -0.5927734375, -0.495849609375, -0.1744384765625, 0.400634765625, -0.22119140625, -0.93896484375, -1.0400390625, 1.2509765625, -0.64453125, 0.0016202926635742188, -0.55517578125, -1.3056640625, -0.134521484375, -1.4404296875, -1.78515625, -0.444580078125, -1.1201171875, -1.5625, -0.71826171875, -1.61328125, 0.413330078125, -2.53515625, -0.76953125, -0.7470703125, -1.3466796875, 0.28857421875, -1.15234375, -0.491943359375, 0.7861328125, -0.2452392578125, 1.5986328125, -0.2467041015625, 0.27392578125, 1.2197265625, -1.65625, 0.07171630859375, 0.473876953125, -1.68359375, 1.2744140625, -0.412109375, 1.45703125, 0.308837890625, 0.10931396484375, -0.0167694091796875, -1.9609375, 0.2039794921875, 0.8583984375, 0.284423828125, 0.66064453125, -0.403076171875, 0.79150390625, 0.09234619140625, -1.328125, -0.48681640625, -2.583984375, -1.640625, 0.384521484375, 0.261962890625, -0.8173828125, 0.7392578125, 0.52490234375, -0.7041015625, -0.169189453125, 1.626953125, -0.3935546875, -1.681640625, 0.42724609375, -1.751953125, -0.33154296875, 0.05426025390625, -0.99462890625, 0.00276947021484375, 1.2705078125, -0.52490234375, -2.248046875, 1.9072265625, -1.453125, 1.1103515625, -1.5751953125, 0.04205322265625, 0.252197265625, -0.8408203125, 0.51025390625, 0.06951904296875, -0.27294921875, -2.029296875, 0.3740234375, 1.658203125, -2.328125, 0.53759765625, 0.329345703125, 1.5419921875, 1.2666015625, -1.6826171875, 2.41796875, -0.04473876953125, 0.73779296875, -1.21484375, -0.66064453125, 0.406982421875, 1.4541015625, -1.6142578125, -1.59765625, 0.2083740234375, -0.78857421875, 0.77880859375, -0.181396484375, -0.55517578125, 1.2626953125, 0.79248046875, 1.6162109375, 0.5107421875, -0.33837890625, -0.105712890625, -0.712890625, 0.022247314453125, 0.62548828125, 0.393310546875, -0.43310546875, 1.224609375, 0.7841796875, 1.5185546875, 0.21630859375, 0.99072265625, 0.82177734375, 2.51953125, -0.607421875, -0.14306640625, 0.87646484375, -0.240234375, 0.8232421875, 0.6337890625, -0.73681640625, 1.064453125, -1.189453125, -0.51025390625, -1.375, -0.10540771484375, -0.59228515625, -1.3828125, 0.93017578125, -0.640625, 0.1651611328125, -0.0438232421875, 0.96435546875, -0.2490234375, 0.6181640625, 3.21484375, 1.1845703125, -1.8564453125, -0.002841949462890625, 1.2890625, 0.66259765625, 0.87841796875, -1.4853515625, -0.5341796875, 1.3544921875, 0.11187744140625, -0.828125, 1.486328125, 0.44775390625, 1.775390625, -1.3232421875, -1.130859375, -0.5576171875, 0.271484375, -1.43359375, -0.08807373046875, -1.0458984375, 0.68115234375, 1.046875, -0.76025390625, -0.65625, -0.2149658203125, -0.0421142578125, -0.7548828125, 0.8037109375, 0.9140625, -0.6142578125, -1.1064453125, -0.1357421875, 0.47216796875, -0.76318359375, -1.2353515625, -1.017578125, 0.373291015625, 0.400634765625, -0.388916015625, -0.57470703125, -2.34375, 0.360595703125, 0.284912109375, 1.466796875, 0.025146484375, 0.89794921875, 0.92041015625, 0.87109375, -0.5068359375, -0.76611328125, -1.0068359375, -0.403076171875, 0.630859375, 0.9150390625, -0.5810546875, 0.0038604736328125, -0.71533203125, 0.74267578125, 1.90625, -0.1903076171875, -0.57861328125, 0.0025844573974609375, 1.1103515625, 0.486083984375, -1.3466796875, 0.485107421875, -0.62451171875, -1.0595703125, -0.9775390625, 1.361328125, -0.8935546875, -1.0302734375, -0.18310546875, 0.427490234375, 0.1099853515625, -1.35546875, -0.480224609375, 1.34765625, -0.08062744140625, -0.66748046875, -0.99365234375, -1.181640625, -0.69384765625, -0.34619140625, 0.42822265625, 0.634765625, 0.890625, -1.7333984375, -1.0, -1.337890625, 1.5166015625, 1.1787109375, 0.541015625, -1.0888671875, 0.84716796875, 0.4384765625, -0.282958984375, 0.293701171875, -0.0843505859375, -1.447265625, 0.455810546875, -0.389892578125, 1.2958984375, -1.146484375, -0.44970703125, 1.353515625, 0.1805419921875, -1.2216796875, -2.072265625, 0.165283203125, 1.708984375, 0.457763671875, -0.06793212890625, -0.37255859375, 1.0830078125, 0.8486328125, -0.47314453125, 0.10882568359375, -1.8427734375, 0.5244140625, -0.04180908203125, 0.82763671875, -0.6689453125, -0.56591796875, -0.78662109375, -0.97998046875, -0.401611328125, -2.15625, 0.196044921875, -0.09869384765625, 1.5869140625, -0.99658203125, 0.217041015625, 0.462890625, 0.392333984375, -0.09814453125, 0.759765625, -1.42578125, -0.71240234375, 0.2078857421875, -0.10382080078125, -0.541015625, -0.56640625, -1.234375, 0.485595703125, 0.467529296875, -0.08526611328125, 0.197998046875, -1.03125, -1.46875, -0.5654296875, -0.10614013671875, 0.9853515625, -0.9482421875, -0.447265625, 0.285400390625, 0.50244140625, 0.343994140625, 1.3701171875, 0.791015625, 0.285400390625, -0.88818359375, -2.484375, -0.91943359375, 0.1981201171875, 0.1805419921875, -1.8310546875, 0.03448486328125, 0.473388671875, 0.38916015625, -1.9873046875, 0.326904296875, 1.0205078125, 0.273193359375, 0.8056640625, -0.001773834228515625, -1.2294921875, -0.9921875, 1.505859375, -1.0498046875, 0.407470703125, 2.755859375, -1.4013671875, -1.9970703125, 0.324462890625, 0.505859375, 0.059539794921875, 1.033203125, 0.59765625, 0.159423828125, -0.56591796875, 0.8486328125, 0.52685546875, -0.63525390625, -0.69091796875, -0.0159149169921875, 0.2041015625, 0.3056640625, -0.3251953125, -0.468994140625, -1.822265625, -0.38330078125, -0.394775390625, -1.0146484375, 1.7197265625, 1.3515625, -1.068359375, -0.1693115234375, -1.2626953125, -0.399169921875, 2.640625, -0.057342529296875, 1.619140625, 1.5263671875, -0.0582275390625, 1.5380859375, -0.61962890625, -0.5703125, 0.87548828125, 1.92578125, 0.70751953125, 0.55078125, -0.9931640625, 0.33935546875, 0.26416015625, 0.6611328125, -0.249755859375, 0.354736328125, -0.4716796875, 0.568359375, 0.08184814453125, -0.42041015625, 1.7626953125, -0.3203125, -1.013671875, -0.64111328125, -2.208984375, 0.065673828125, -1.619140625, 0.381103515625, 0.1705322265625, -1.0419921875, 0.6318359375, 0.255126953125, -0.80712890625, 0.5625, 0.9970703125, -0.1595458984375, -0.6025390625, -0.86376953125, 0.4052734375, 2.40625, -1.775390625, 0.428955078125, 0.404541015625, -1.05859375, 1.3623046875, 0.164306640625, 0.1317138671875, -0.035491943359375, 0.027496337890625, 0.65185546875, 2.19921875, 0.36865234375, 0.92724609375, 0.0958251953125, -0.3857421875, -0.350830078125, 0.59912109375, 0.55029296875, -1.6689453125, -0.172119140625, 1.224609375, 0.560546875, 0.1397705078125, 2.748046875, -0.48876953125, 0.22802734375, -0.17529296875, -0.3681640625, -0.646484375, -0.32177734375, -0.164794921875, -0.340087890625, -1.431640625, -1.37890625, -0.32275390625, 1.4677734375, 0.0257415771484375, -0.28857421875, -1.111328125, 0.52392578125, -0.56201171875, 1.236328125, -0.390625, 0.50830078125, 1.2060546875, 1.205078125, 0.6337890625, -1.423828125, -0.8447265625, -0.0732421875, -0.83251953125, -2.396484375, 0.74560546875, -1.1572265625, 0.1591796875, -1.6748046875, -0.67822265625, 0.6171875, -0.912109375, -1.068359375, 2.03515625, 0.03118896484375, -0.8798828125, 0.6416015625, -0.295166015625, -0.341796875, -1.1240234375, -0.59619140625, 0.78955078125, 0.10235595703125, -0.55224609375, -3.392578125, 1.701171875, -0.5703125, 0.76953125, 2.0234375, -1.380859375, -0.271728515625, -1.419921875, -1.2568359375, -1.337890625, -0.60986328125, -0.05126953125, 1.3505859375, 0.445068359375, -0.0689697265625, -1.0791015625, -0.176513671875, -0.5078125, 0.962890625, 0.425537109375, 0.68115234375, 1.685546875, 1.140625, -0.5341796875, 0.5283203125, 0.5810546875, -0.50732421875, 1.1435546875, -0.082763671875, 0.59619140625, 0.1092529296875, -1.181640625, -0.2479248046875, 0.86572265625, 0.433837890625, -0.83544921875, -0.3662109375, -0.183349609375, 0.08184814453125, -0.11700439453125, 0.173095703125, -0.5576171875, -0.6875, 1.96484375, -1.1748046875, -1.0615234375, 0.0804443359375, -0.0899658203125, -0.5439453125, 1.0341796875, 0.75830078125, 0.0826416015625, -1.6533203125, 1.16015625, 1.9052734375, 0.36181640625, 1.0185546875, -0.499267578125, -0.284912109375, -0.5478515625, 0.49755859375, 0.73046875, -0.9541015625, 0.64697265625, 1.822265625, -0.0302276611328125, 0.437744140625, -0.52197265625, -2.673828125, 0.97900390625, 0.068115234375, 1.6513671875, 0.46435546875, 0.5732421875, -1.822265625, -0.51904296875, -1.931640625, 1.6728515625, -2.40234375, -0.568359375, 0.2099609375, 1.2626953125, 0.0265960693359375, 1.0185546875, 1.5673828125, -0.300537109375, 0.326416015625, -0.65625, -1.7734375, 1.6474609375, -0.08331298828125, -0.397705078125, 1.841796875, 0.267578125, -1.236328125, 0.6513671875, 0.1751708984375, 1.1376953125, -0.78369140625, -0.1302490234375, -0.419677734375, 0.37939453125, 1.056640625, 1.013671875, 0.489013671875, 0.91650390625, -0.392822265625, 0.7763671875, 2.01171875, 0.9873046875, 1.46484375, -1.154296875, 1.283203125, -0.158203125, 0.98681640625, -0.66943359375, -1.0810546875, 0.390380859375, -0.09515380859375, 0.89697265625, -1.0146484375, -1.138671875, 1.0810546875, 0.365234375, -2.228515625, 0.019439697265625, 0.87255859375, -0.074951171875, 1.0048828125, 0.36572265625, -0.298828125, 0.353271484375, 0.97216796875, 0.724609375, -1.1650390625, 1.5224609375, -0.274658203125, 0.96533203125, -0.1795654296875, 0.0276641845703125, 0.0496826171875, 0.51220703125, -0.5185546875, 1.556640625, -0.99267578125, 0.188232421875, 0.83251953125, -0.7314453125, -0.58251953125, 0.1636962890625, -0.25439453125, 0.673828125, -0.18310546875, 0.71533203125, 0.239990234375, 0.83447265625, 0.10211181640625, 0.53369140625, -0.296630859375, -0.42431640625, -1.3798828125, 1.568359375, -0.5029296875, 1.2109375, 1.12109375, -0.423583984375, -1.1533203125, 0.375, -0.317626953125, 0.966796875, 0.3837890625, -0.88232421875, 0.1873779296875, 1.23828125, 0.58251953125, -0.0535888671875, 0.607421875, -0.71484375, 0.432861328125, -1.61328125, 1.3330078125, 0.166259765625, 2.962890625, 0.15576171875, 1.7646484375, -0.167724609375, -2.455078125, 0.067138671875, -2.740234375, -0.45654296875, 0.59521484375, -0.25439453125, 0.36962890625, -0.626953125, 0.0124359130859375, -1.4462890625, -0.447509765625, 0.322265625, 1.8681640625, -0.36376953125, -0.27978515625, -0.9375, -0.568359375, -0.45849609375, 0.98876953125, 0.412109375, -0.92138671875, 0.289794921875, -0.5888671875, -0.431884765625, 0.60595703125, 1.4306640625, -0.195068359375, 0.445556640625, 0.79443359375, -0.845703125, 0.2071533203125, -1.173828125, -0.79541015625, 0.95703125, 1.15234375, -0.235595703125, -1.33984375, 1.5966796875, 0.7060546875, -0.42919921875, -1.0, -0.6572265625, 0.68994140625, -1.5908203125, -0.60205078125, -0.96728515625, -0.1533203125, 0.39111328125, -1.271484375, 0.445068359375, 0.9580078125, 0.70458984375, -0.59033203125, -0.501953125, 0.1572265625, -1.2255859375, 1.51171875, 0.1962890625, -0.6826171875, 0.36474609375, -0.43408203125, -0.97119140625, 0.69287109375, -0.6298828125, -2.890625, -1.0595703125, 0.270263671875, 1.1953125, 0.67041015625, -1.0078125, 0.75634765625, -0.7421875, 2.2734375, 0.3916015625, 1.3583984375, 0.1639404296875, -1.3369140625, -0.375732421875, -0.6015625, -1.068359375, -1.7548828125, 1.06640625, 0.6162109375, 0.309814453125, -2.552734375, 0.54052734375, 0.482177734375, -0.22314453125, -0.86474609375, -0.65380859375, -1.6298828125, -0.3671875, 0.372314453125, -0.02667236328125, -0.97265625, 0.0987548828125, 0.81201171875, 1.3486328125, 1.5625, -1.1787109375, 0.482421875, -1.7744140625, 1.3212890625, 1.1875, 0.192626953125, -1.0986328125, 0.9638671875, 0.81787109375, -0.023040771484375, 1.09375, -0.29345703125, 0.1595458984375, -0.2015380859375, -0.37939453125, -2.080078125, -1.1640625, 0.04620361328125, 0.97998046875, 1.728515625, -0.0771484375, -0.08270263671875, -0.8359375, 0.189208984375, 0.71435546875, 0.70361328125, 0.052154541015625, -1.5673828125, -1.8203125, -0.87890625, -1.306640625, -1.3701171875, -0.69140625, 1.2275390625, 1.6630859375, -0.225830078125, 0.12017822265625, 2.5078125, -1.3798828125, 0.5341796875, -0.36962890625, 0.76806640625, -0.5048828125, -0.9619140625, 1.2861328125, 0.66748046875, -1.7763671875, -0.76025390625, 0.50634765625, 0.9306640625, 0.2744140625, 0.11883544921875, 0.87158203125, -0.434814453125, 0.00481414794921875, -1.453125, -0.10455322265625, 1.0048828125, -1.01953125, 0.79296875, 0.33447265625, 1.4853515625, -0.8876953125, 0.302734375, -0.6337890625, -0.018035888671875, -1.2880859375, -0.30029296875, 1.080078125, 0.3935546875, 1.2626953125, 0.2183837890625, 0.28564453125, 0.334716796875, -0.40478515625, -0.77294921875, -0.1705322265625, -0.64208984375, 0.3486328125, 1.24609375, -0.154541015625, 0.65478515625, -0.1328125, -0.65185546875, -0.272216796875, 0.79150390625, -1.068359375, 1.47265625, -0.6943359375, 0.7841796875, 1.4169921875, 1.8896484375, -0.309326171875, 0.78271484375, -0.919921875, -0.37548828125, -0.1190185546875, -1.984375, 0.5830078125, 0.075439453125, 0.1512451171875, 0.2435302734375, -0.52392578125, 2.2890625, 1.9580078125, 0.07293701171875, -0.7607421875, -0.76806640625, 1.298828125, -1.6103515625, -1.5, 0.72265625, 1.2412109375, -0.73779296875, -0.72412109375, 0.83544921875, -0.07110595703125, -0.35205078125, 0.444580078125, 0.4931640625, -0.459228515625, -1.0341796875, -0.290771484375, 0.9140625, -0.441650390625, -0.0341796875, -1.4775390625, 2.296875, 1.62109375, 0.0994873046875, 1.3232421875, 0.88330078125, 1.1748046875, -0.1611328125, -0.541015625, -0.337890625, -0.1881103515625, -0.80712890625, 1.302734375, 0.94140625, -0.2060546875, -3.189453125, 1.53515625, -1.0390625, 0.5380859375, -0.919921875, 1.1943359375, -0.96044921875, 1.1962890625, 0.59912109375, 0.10272216796875, 0.429443359375, 1.8779296875, -0.98583984375, -1.0322265625, -0.4501953125, -1.64453125, 0.364990234375, 0.0960693359375, 0.39794921875, -2.2421875, -2.095703125, 1.5068359375, -0.239501953125, -1.896484375, -0.7890625, 2.482421875, 0.5537109375, 1.6650390625, 0.896484375, -0.159423828125, 0.509765625, 0.10516357421875, 1.724609375, 0.8046875, 0.8857421875, -1.142578125, 0.8798828125, -0.468505859375, 3.45703125, -1.2451171875, -1.00390625, -0.11639404296875, 1.294921875, -0.56982421875, -0.1961669921875, 0.379638671875, -1.0595703125, -3.181640625, 2.8828125, 0.48291015625, -1.337890625, -0.0018129348754882812, -1.12109375, 1.5458984375, -0.335693359375, 0.80712890625, 0.6953125, -0.697265625, -0.2744140625, 0.052001953125, -0.3466796875, -1.275390625, -1.845703125, -2.345703125, -0.8212890625, -0.3583984375, -0.33837890625, 0.1055908203125, 0.20947265625, -0.9814453125, 1.2236328125, 0.045074462890625, 0.7197265625, 0.462158203125, -0.55859375, 0.31298828125, 0.71142578125, -0.46240234375, -0.50439453125, 0.276123046875, -0.4033203125, -0.98876953125, -1.6669921875, 1.87890625, -0.290771484375, -1.2109375, 0.298095703125, 0.034698486328125, 0.1707763671875, -0.04412841796875, -0.201416015625, 1.333984375, -0.06622314453125, -0.35107421875, -0.3544921875, 0.171142578125, -0.220703125, 0.73681640625, -0.9697265625, 0.81982421875, 1.0869140625, -2.24609375, -1.05078125, -0.1591796875, -0.64404296875, 0.9287109375, 0.9775390625, -0.295166015625, -0.09027099609375, -1.25, -0.35693359375, 1.9287109375, -0.149658203125, 0.93017578125, -0.4326171875, -0.299560546875, -0.56787109375, -1.5712890625, 0.6484375, 0.0914306640625, 1.71875, 2.6328125, -0.316650390625, 0.315673828125, -1.4873046875, 0.293701171875, 0.5791015625, 0.459228515625, 0.9306640625, -0.50537109375, -0.1500244140625, -1.2548828125, -0.410400390625, 0.34716796875, 2.83203125, 0.51318359375, 0.60205078125, 1.203125, -0.60107421875, -0.994140625, -0.9365234375, -0.4443359375, 0.736328125, 0.681640625, -0.157470703125, -0.10113525390625, -0.9130859375, 0.75537109375, -0.2037353515625, 0.00432586669921875, -0.5732421875, 0.88037109375, 1.955078125, 1.0458984375, 0.1383056640625, -0.6982421875, -2.04296875, 0.07672119140625, -0.1734619140625, -0.609375, 0.048797607421875, -0.330322265625, -0.51318359375, 0.0545654296875, 0.42578125, 1.8291015625, -1.294921875, -0.1812744140625, 0.36083984375, 0.2042236328125, 0.436279296875, -0.488525390625, 0.6826171875, -0.98193359375, -0.64404296875, 0.57275390625, -1.6640625, -0.900390625, -0.34228515625, -0.00013136863708496094, 1.123046875, 1.0, 0.313720703125, -0.057647705078125, 1.11328125, 0.66259765625, -1.1875, -0.36865234375, 0.85888671875, -0.20751953125, -0.064208984375, 1.2880859375, 0.09283447265625, 0.220703125, -1.890625, -0.26806640625, 0.7099609375, 1.0400390625, -1.435546875, 0.716796875, -0.86767578125, -1.1142578125, 0.014617919921875, 0.313232421875, -0.87353515625, -0.2301025390625, 0.0308990478515625, -1.9150390625, 1.15625, -1.3896484375, 0.61669921875, -0.705078125, -0.0823974609375, 0.3056640625, -0.4521484375, -0.215087890625, 0.90625, -0.21875, -0.529296875, 2.07421875, 2.154296875, -1.080078125, 1.2490234375, -0.17236328125, -1.1064453125, -0.422607421875, 0.634765625, -1.935546875, 0.3359375, 0.478515625, 1.2939453125, -0.93505859375, 0.9482421875, -0.09130859375, 0.80859375, -1.056640625, 1.0078125, 0.388671875, 1.2333984375, -1.3037109375, 0.11956787109375, -0.10015869140625, 0.38427734375, -0.338623046875, 0.373291015625, 0.69140625, 0.88623046875, -0.4658203125, -1.4228515625, 0.2039794921875, 1.40625, -0.0340576171875, 0.498291015625, 0.33203125, 0.5751953125, 0.10699462890625, -0.369384765625, 0.07086181640625, 0.34521484375, 0.006069183349609375, 1.4453125, -0.59033203125, -0.84814453125, 1.4326171875, 2.064453125, 1.62890625, 0.97119140625, 0.51708984375, -1.6171875, 0.6865234375, -1.6552734375, -0.291748046875, 0.87646484375, 0.03985595703125, -0.6220703125, 0.02490234375, -0.454345703125, 0.36376953125, 0.6455078125, 2.119140625, -0.9326171875, 0.421630859375, -0.00681304931640625, -0.90185546875, 0.89697265625, -1.271484375, -0.01177215576171875, 0.861328125, -0.4091796875, 0.0221405029296875, -0.4287109375, -0.55908203125, 0.75537109375, 0.84716796875, -0.5947265625, 0.0219268798828125, -0.15869140625, -0.6064453125, 1.6572265625, 1.2724609375, -0.52734375, 1.283203125, 0.421630859375, -0.374755859375, -0.8125, -1.1015625, -0.671875, 0.6875, -0.045654296875, -1.0380859375, 0.1451416015625, 0.1339111328125, 0.03521728515625, 0.50390625, 1.4013671875, 1.7490234375, 0.00908660888671875, 0.615234375, -1.4931640625, 0.34130859375, -0.106201171875, -0.39697265625, -0.1505126953125, 1.62890625, 0.71826171875, -1.041015625, 0.5556640625, 0.541015625, 2.45703125, 1.08203125, 0.78466796875, -0.3515625, 1.1923828125, -0.1817626953125, -1.2353515625, 0.326904296875, 1.1435546875, -0.98876953125, -1.4658203125, -0.1259765625, 1.7724609375, -0.059356689453125, 0.0291900634765625, 2.380859375, -0.1776123046875, 0.341796875, -1.2841796875, 2.49609375, 2.263671875, -0.9892578125, 1.8388671875, -0.456787109375, 1.537109375, 0.64013671875, 2.51171875, -1.353515625, 0.76123046875, -0.35009765625, -1.603515625, 0.65234375, -0.344482421875, 0.9208984375, 0.8447265625, 1.140625, 1.3642578125, 0.6640625, 0.01216888427734375, 1.2783203125, -1.2978515625, 1.0078125, 1.3271484375, -0.32763671875, -2.919921875, -0.76416015625, -0.60400390625, -1.2451171875, -1.9208984375, 0.31884765625, -1.6884765625, -1.205078125, 0.1978759765625, 0.65771484375, -0.796875, -0.55126953125, 2.126953125, 0.361083984375, -0.71337890625, -0.5791015625, -1.0107421875, 0.29833984375, -1.611328125, -0.81982421875, 0.74755859375, -0.15478515625, 0.7265625, 1.8056640625, 1.6259765625, 0.556640625, 0.5634765625, -0.1949462890625, -0.322265625, -1.15234375, -0.9423828125, -0.04132080078125, 0.3310546875, 1.3125, -0.07989501953125, -1.7705078125, 0.2073974609375, 0.09722900390625, 1.939453125, 0.9482421875, 2.03125, 0.6572265625, -0.36669921875, -0.56787109375, -1.5419921875, 0.80712890625, 0.3076171875, -0.21337890625, -0.76171875, -1.59765625, 0.5087890625, -0.1436767578125, -0.432861328125, -1.0810546875, 1.6328125, 1.1923828125, -0.0201873779296875, -0.48828125, 0.388671875, -0.29345703125, -1.166015625, 0.1328125, -0.38232421875, -0.83740234375, -0.186279296875, -0.353515625, -0.8193359375, 0.3056640625, 2.048828125, -0.73681640625, 1.291015625, -0.06451416015625, -0.91796875, 0.18798828125, -0.32763671875, 0.416748046875, -1.73046875, -0.10992431640625, -0.339599609375, 0.9833984375, 0.73095703125, -1.173828125, 0.4287109375, 0.2181396484375, 0.6015625, -1.654296875, -0.5576171875, 0.86474609375, -0.89208984375, 0.556640625, -0.18896484375, 0.580078125, 0.85107421875, -0.82421875, -0.18994140625, -0.7607421875, 0.51806640625, -0.572265625, -0.8408203125, 0.705078125, -0.8955078125, -1.408203125, 0.0217437744140625, -0.048919677734375, 0.3154296875, -0.4931640625, 0.10858154296875, 0.11407470703125, 1.7001953125, -0.484619140625, 0.1849365234375, -0.09674072265625, -0.1614990234375, 0.018829345703125, -0.60595703125, -0.447509765625, 1.890625, -1.8779296875, -0.98486328125, 1.2802734375, -2.5, -0.21240234375, -1.244140625, 0.8984375, 2.171875, 1.2587890625, 1.9072265625, -1.5537109375, 0.4189453125, -0.54345703125, 0.292724609375, -0.51611328125, 0.131591796875, 0.260009765625, 0.53173828125, 0.272705078125, -0.4619140625, -0.85888671875, 0.0650634765625, 0.6025390625, 0.1544189453125, 0.513671875, 1.0595703125, -0.76025390625, 0.38818359375, 0.10394287109375, 1.0625, 0.52197265625, -3.037109375, 0.57958984375, 0.487060546875, -0.30322265625, 1.4462890625, 0.21630859375, 0.132568359375, 1.203125, -0.467041015625, 0.403076171875, 1.8876953125, -0.92138671875, -0.5458984375, 0.55908203125, 0.147216796875, 0.97216796875, 0.712890625, 0.090576171875, 0.45751953125, 0.88134765625, 1.107421875, 0.2607421875, -0.8505859375, -0.41796875, -2.392578125, 0.06890869140625, -0.260986328125, -1.548828125, -1.2705078125, -0.74951171875, -0.5947265625, -0.51318359375, 0.55712890625, -0.9833984375, 0.8564453125, -1.1025390625, -0.74560546875, -0.9951171875, 0.0699462890625, -0.8876953125, -0.92822265625, 1.7021484375, -1.5537109375, -1.2490234375, 0.495849609375, 1.2080078125, 0.10552978515625, -0.8623046875, 0.99267578125, 0.334228515625, 0.0250396728515625, 0.88623046875, -2.384765625, 0.2783203125, -0.9599609375, -0.55615234375, -1.865234375, -0.720703125, -1.1044921875, -0.6875, 1.0556640625, 0.2474365234375, -0.84423828125, 0.83447265625, 0.79345703125, 0.6953125, -0.60107421875, -0.267333984375, -0.65869140625, 0.5478515625, -0.2476806640625, 0.83544921875, -0.061187744140625, -1.1962890625, -0.300048828125, 0.59716796875, 0.58251953125, 0.5537109375, 0.2900390625, 0.062103271484375, 0.77978515625, -0.089111328125, -0.454345703125, 0.288818359375, 0.0679931640625, -0.049163818359375, 0.03204345703125, 0.1427001953125, -0.37548828125, -1.9521484375, 0.4345703125, -0.7197265625, 0.509765625, -1.498046875, 0.36279296875, -0.151611328125, 0.0841064453125, 0.38232421875, -0.042877197265625, 0.71875, 0.18359375, 0.6494140625, -0.015167236328125, -1.3603515625, -2.6015625, 0.4228515625, -1.7763671875, -0.415283203125, -1.0205078125, -1.60546875, 0.351806640625, -0.11090087890625, 0.10882568359375, 0.2034912109375, -0.919921875, 0.371826171875, 0.1346435546875, 0.5625, 0.16845703125, 0.1422119140625, -0.216064453125, 0.83544921875, 0.4658203125, -0.1810302734375, -1.1015625, -0.89794921875, 1.52734375, 1.4462890625, -0.1282958984375, -0.287841796875, 0.101806640625, 0.56640625, -1.26953125, 0.86669921875, -1.5390625, -0.533203125, 0.39306640625, -1.6875, 1.4052734375, 0.61328125, -0.8271484375, 0.2353515625, 0.63232421875, -1.62890625, 0.89453125, -0.2177734375, 1.83203125, 1.78125, -0.763671875, 1.01953125, 0.927734375, -1.0595703125, -1.7724609375, 0.79052734375, -0.43212890625, -0.255126953125, 0.058868408203125, 1.025390625, -1.015625, -0.249267578125, 0.01197052001953125, -0.229248046875, 0.048248291015625, 1.9140625, 0.4658203125, -0.7236328125, -0.662109375, 1.0830078125, -0.56640625, -0.78369140625, 1.3876953125, 1.869140625, -0.294921875, 1.4140625, 0.042694091796875, -0.490478515625, 0.90283203125, 0.9658203125, -0.6171875, -0.41064453125, 1.294921875, -0.337890625, 0.291748046875, 0.28857421875, 0.356201171875, 1.0498046875, -0.4140625, 0.71728515625, 0.9755859375, 0.1756591796875, 1.4345703125, -0.331298828125, 1.51171875, -2.822265625, -1.234375, -1.7060546875, 0.79833984375, 0.859375, 0.1617431640625, 0.0682373046875, 0.2061767578125, 0.36083984375, -0.6376953125, 0.62744140625, -2.140625, -1.40234375, -1.0908203125, -0.78955078125, -1.5732421875, -1.1640625, 0.429931640625, -0.93408203125, -0.0135650634765625, -1.2373046875, -0.396240234375, -0.0948486328125, -0.564453125, -1.1435546875, -0.88525390625, 0.333251953125, 0.6650390625, 0.9541015625, -0.42578125, -0.34619140625, -0.58544921875, 1.36328125, 0.6689453125, -0.51318359375, 0.293212890625, -1.2939453125, -0.9287109375, 0.5712890625, -0.3525390625, 0.373291015625, -0.869140625, -0.60009765625, -0.501953125, -0.51025390625, -0.1409912109375, 0.90380859375, -0.302490234375, 2.427734375, -0.7685546875, -0.268798828125, 1.193359375, -1.8935546875, 0.63525390625, 0.336181640625, 0.119140625, 0.5908203125, -0.4892578125, -0.2467041015625, 0.6904296875, 0.0097503662109375, 0.7353515625, -0.09063720703125, 0.6015625, -1.1259765625, -1.173828125, 1.228515625, 0.59814453125, -2.142578125, -0.374267578125, 1.154296875, 0.57666015625, 0.318359375, 0.8642578125, -0.0538330078125, 0.08294677734375, 1.123046875, -1.330078125, 1.091796875, -1.859375, -1.0537109375, -0.058349609375, 0.7255859375, 0.41064453125, 0.70654296875, -1.8916015625, -0.12420654296875, 0.58251953125, -0.468017578125, 0.1884765625, 0.05169677734375, -0.703125, -0.84814453125, -0.015045166015625, -0.345458984375, 0.0027828216552734375, 0.97314453125, -0.11138916015625, -1.208984375, 0.97119140625, -0.1566162109375, 0.86865234375, -0.9853515625, 0.6787109375, -2.10546875, -0.468505859375, 1.771484375, -1.736328125, -1.7998046875, -1.5830078125, -0.315673828125, 0.9033203125, 0.76904296875, -1.0595703125, 2.40234375, -0.78857421875, 1.7373046875, -0.255126953125, 1.115234375, 0.01343536376953125, -0.375244140625, -0.336181640625, -0.4638671875, -0.54345703125, 1.1064453125, 0.18603515625, -0.034423828125, 1.134765625, -0.1180419921875, 0.457763671875, -1.447265625, -0.86669921875, 1.154296875, 0.5380859375, 1.66796875, 0.1602783203125, 0.38232421875, -1.0068359375, 0.453369140625, -0.20068359375, -0.58349609375, 0.50341796875, -0.5654296875, 0.73291015625, 1.2431640625, 1.3076171875, 0.4365234375, 1.4990234375, -0.1842041015625, -0.69482421875, 0.50439453125, 1.2119140625, 0.4443359375, 1.5830078125, 1.322265625, 0.91650390625, 0.4150390625, -0.72412109375, -3.21875, 1.7783203125, 2.078125, 0.94921875, -1.51171875, -0.2222900390625, -0.78955078125, -0.5458984375, -0.0259857177734375, -0.04010009765625, 1.0625, -0.0516357421875, -0.228271484375, 2.4921875, 0.439697265625, -0.8359375, -1.1845703125, -0.96484375, -1.0517578125, 1.4150390625, 1.173828125, -2.14453125, -0.853515625, -0.447509765625, 0.0977783203125, -0.462646484375, 0.5634765625, 1.4775390625, 0.6044921875, -1.5927734375, -0.2822265625, -0.68896484375, 1.447265625, -0.05535888671875, -2.841796875, -2.359375, 1.5595703125, 2.080078125, -1.4658203125, 0.333740234375, 0.0606689453125, 0.1356201171875, -1.8525390625, -0.02099609375, -0.28515625, -0.41064453125, -1.375, -0.6943359375, -0.521484375, -1.7470703125, -1.607421875, 0.07720947265625, -1.115234375, -1.59375, -0.2008056640625, -0.92919921875, 0.58447265625, -0.0928955078125, -0.884765625, -0.9140625, 1.6201171875, -0.1512451171875, -0.27587890625, -1.447265625, -0.55078125, 0.367919921875, -0.58984375, 0.41845703125, -0.9580078125, 0.440673828125, 0.384033203125, 0.2476806640625, -0.8525390625, 0.97900390625, 0.744140625, 0.61962890625, -0.1602783203125, -0.349365234375, 1.1533203125, 0.4375, 0.59326171875, -0.7333984375, 0.2337646484375, -1.7998046875, -0.08489990234375, 1.0751953125, -0.921875, -0.439208984375, 0.21875, 0.394287109375, 0.005359649658203125, 0.64599609375, -0.3720703125, -0.4921875, -0.485595703125, -0.07708740234375, 0.88232421875, 1.2470703125, -0.2305908203125, -1.4619140625, -0.6689453125, 2.115234375, -0.329833984375, 1.32421875, 1.2978515625, -0.271728515625, -3.095703125, -0.0667724609375, -0.80029296875, 0.2305908203125, -0.30810546875, 0.1607666015625, 0.6875, -1.318359375, 0.06439208984375, 1.48046875, 3.818359375, 1.556640625, -0.654296875, -0.068115234375, 0.5830078125, 0.85986328125, -0.43994140625, -0.7119140625, 0.5986328125, -0.9228515625, 0.1495361328125, 0.8232421875, 1.8193359375, -0.28857421875, 1.3544921875, -0.64794921875, 1.189453125, 0.228271484375, 0.10333251953125, 1.3271484375, 0.265380859375, -0.37841796875, -0.56103515625, -0.398681640625, 0.314208984375, 1.248046875, 0.11163330078125, -0.03985595703125, -1.171875, 1.255859375, -0.430419921875, 0.8779296875, -0.9951171875, -0.281494140625, -1.560546875, -0.4560546875, 0.203125, 0.1502685546875, -0.62890625, 0.03997802734375, -1.3046875, 0.49755859375, 1.18359375, 1.1689453125, -1.38671875, -0.318115234375, -0.355712890625, -0.11090087890625, 0.66015625, -0.6025390625, -0.365966796875, -1.0078125, 0.69140625, 0.490966796875, 0.7734375, 0.47900390625, -1.63671875, -0.370361328125, -0.45751953125, 1.90625, -3.302734375, 0.3515625, 0.0887451171875, 0.716796875, 0.76025390625, 1.548828125, 1.509765625, -0.85205078125, 1.8984375, -0.320556640625, -0.62890625, -0.326904296875, -1.037109375, -1.125, -0.06744384765625, 1.09375, 0.78076171875, 0.83056640625, -0.73974609375, -1.041015625, 0.305419921875, 0.28076171875, 3.47265625, 0.30517578125, 0.18701171875, -1.1640625, 2.39453125, 0.374755859375, 2.015625, -0.485595703125, -1.5615234375, -0.68603515625, -0.2054443359375, 0.74072265625, -0.055389404296875, 0.498046875, 0.828125, -0.127197265625, 0.0458984375, 0.90625, 0.0090179443359375, 0.716796875, -1.0205078125, 0.82470703125, -0.370361328125, 0.25634765625, -0.250732421875, 0.6005859375, 0.396484375, 1.6083984375, 0.39404296875, -1.7919921875, -0.4814453125, -1.265625, 1.7431640625, 0.81787109375, -0.393798828125, 0.59423828125, 0.1881103515625, 1.1767578125, 0.865234375, -0.6279296875, 0.81884765625, 0.1292724609375, -0.5810546875, -0.54150390625, -0.46044921875, -0.124267578125, -1.0234375, -0.020294189453125, 0.89990234375, 0.67822265625, 0.403564453125, 0.8271484375, 1.2177734375, 0.308837890625, -0.5341796875, -0.15283203125, -1.2177734375, 0.458740234375, -0.0941162109375, -0.79736328125, 1.2177734375, 0.5634765625, -0.7265625, 1.3759765625, 1.8623046875, 1.619140625, -1.0146484375, 0.71923828125, 1.755859375, 1.41796875, 1.90625, -0.2607421875, -2.962890625, 0.47412109375, 1.240234375, 0.8115234375, 0.39013671875, -0.9990234375, -1.4033203125, 0.5029296875, 1.2783203125, -0.45751953125, 2.26171875, 0.41357421875, 0.267333984375, -0.576171875, 0.759765625, -0.384765625, -0.291015625, 0.1856689453125, -2.5859375, 0.5791015625, -0.265869140625, 0.68798828125, -0.69140625, 0.2218017578125, 0.6982421875, -0.6142578125, -1.365234375, 0.130126953125, -1.34375, -1.1396484375, 1.919921875, -0.59765625, -0.4853515625, 0.63818359375, -0.31103515625, 0.1629638671875, -1.763671875, 0.197021484375, -1.5126953125, 1.0966796875, 0.640625, 1.2216796875, 3.9375, -0.0034847259521484375, 0.2305908203125, 1.0361328125, -0.0002827644348144531, -0.386962890625, -0.1312255859375, 0.580078125, 0.95947265625, 0.38720703125, -1.12109375, 1.31640625, 1.0927734375, -1.13671875, -1.4453125, 0.146240234375, 1.521484375, 0.5048828125, -0.328857421875, -1.2978515625, -1.357421875, 0.46435546875, -1.4560546875, -2.46484375, -0.142822265625, 0.96484375, 1.3623046875, 0.5341796875, -2.10546875, -0.7919921875, 0.5634765625, -0.916015625, -0.6435546875, -1.3818359375, 0.19873046875, 0.53515625, -1.9921875, 1.1103515625, -0.5, -0.1298828125, -0.6845703125, 0.1920166015625, 0.4501953125, -0.08349609375, 0.83203125, -1.3212890625, 0.2420654296875, -0.1072998046875, 0.64501953125, 0.802734375, -2.033203125, -0.380859375, -0.61376953125, -1.5068359375, 1.044921875, 2.24609375, 0.264892578125, 1.9208984375, -0.22900390625, -0.6689453125, 0.26220703125, 0.74853515625, 0.26708984375, -1.0888671875, -1.861328125, -0.72802734375, -0.501953125, -2.2109375, 0.4130859375, 0.07476806640625, 0.1785888671875, 0.325439453125, 0.4150390625, -0.04443359375, -0.2222900390625, -0.91748046875, -0.078125, 0.1595458984375, -1.0859375, 0.7529296875, 0.31982421875, -0.171875, 1.0048828125, -0.72119140625, 0.88916015625, 1.7392578125, 0.09698486328125, -2.4140625, 0.2080078125, 0.669921875, -0.00762176513671875, 0.46728515625, 0.58203125, -0.505859375, -0.56298828125, -1.017578125, -1.484375, -1.509765625, 1.6171875, 0.4072265625, 1.3515625, 1.4755859375, 1.244140625, -1.9658203125, 0.4658203125, 0.07080078125, 0.365234375, 0.9033203125, 2.267578125, 1.2470703125, 0.11944580078125, 0.040618896484375, 1.6357421875, 1.9423828125, -0.7197265625, -0.8408203125, 1.095703125, 0.477294921875, 0.66845703125, -0.218994140625, -0.341552734375, -0.335693359375, 0.052734375, -1.6416015625, 1.310546875, -0.41845703125, -0.1561279296875, -0.313232421875, -0.006805419921875, 0.701171875, -0.34765625, 0.1834716796875, -1.734375, -1.4169921875, 0.52294921875, -0.25634765625, 0.31298828125, -0.53076171875, -0.2325439453125, 0.259521484375, -0.92333984375, 1.0341796875, 0.2490234375, 1.8203125, -0.418212890625, -0.06341552734375, 0.1165771484375, -0.6181640625, 0.646484375, -0.96923828125, -0.0667724609375, -0.038848876953125, 0.477294921875, -0.919921875, -0.453125, 0.2379150390625, -0.869140625, -0.505859375, 1.5634765625, -0.04315185546875, -0.392822265625, 0.42919921875, -0.2958984375, 0.91162109375, 0.79541015625, -0.8740234375, -0.5302734375, 0.3896484375, 0.7958984375, 0.66064453125, 0.86865234375, 0.0211181640625, 1.29296875, -0.016845703125, -0.02978515625, -0.7646484375, 1.5625, 0.10064697265625, 0.99609375, 0.07769775390625, 1.115234375, -0.98583984375, 0.59228515625, 2.3125, 1.7236328125, 0.415771484375, -0.488037109375, 0.27294921875, 0.07305908203125, 1.2626953125, 0.225830078125, -0.732421875, -0.572265625, -1.1591796875, 0.486572265625, -1.1162109375, 1.28515625, -1.212890625, -1.6689453125, -0.148193359375, 0.0560302734375, -0.81982421875, 1.990234375, 0.34423828125, -0.94775390625, 0.51123046875, -0.27392578125, -0.83349609375, -0.4892578125, 0.93505859375, 1.69140625, -0.401123046875, 1.0791015625, -0.43701171875, -1.490234375, -0.1568603515625, -0.0797119140625, -0.51171875, 0.198974609375, 0.75, -0.26416015625, -0.1690673828125, 0.51708984375, 1.1552734375, 1.8681640625, 1.4033203125, -1.4208984375, -0.283935546875, -2.12890625, 1.0458984375, 0.48583984375, -0.401123046875, 0.257568359375, -0.55224609375, 0.81298828125, 0.5654296875, 0.9482421875, -2.693359375, -2.37890625, 0.5009765625, 0.5537109375, 1.537109375, 0.287109375, -0.52001953125, 0.2283935546875, 1.8037109375, 1.501953125, 0.035186767578125, 1.12109375, -2.0859375, 0.6025390625, -0.75634765625, -1.494140625, -0.708984375, 0.9345703125, 1.84765625, -1.9296875, 0.058990478515625, 0.001842498779296875, -0.548828125, -0.64501953125, 0.56982421875, 0.67626953125, 0.06610107421875, 0.09735107421875, 1.083984375, 0.30517578125, 0.281982421875, -0.54833984375, -2.958984375, -2.091796875, 1.1435546875, 1.9892578125, 0.814453125, -0.6240234375, -1.302734375, -1.3017578125, 2.048828125, 0.87646484375, -0.54443359375, 0.1129150390625, 0.58154296875, 0.99462890625, 1.048828125, 0.4072265625, 0.4384765625, 0.050140380859375, -0.3046875, 0.66748046875, -0.0458984375, -2.1875, 0.876953125, -0.5322265625, 1.0419921875, 0.159423828125, 0.223876953125, -0.2083740234375, 0.54541015625, 1.6181640625, 0.5869140625, 1.3828125, -1.8310546875, 0.7041015625, 0.421142578125, 0.5732421875, -0.2568359375, 0.73193359375, -0.76904296875, -0.04376220703125, -1.49609375, -0.4677734375, -0.63818359375, -0.38232421875, 1.1611328125, -0.478759765625, 0.1253662109375, -0.1451416015625, 1.078125, -0.359619140625, -1.6484375, 0.294189453125, -0.453857421875, -1.626953125, -0.5673828125, 0.33251953125, 1.453125, -0.0648193359375, -1.3017578125, -0.322021484375, -0.1395263671875, -0.80029296875, 0.1982421875, 0.64501953125, -0.2388916015625, -1.544921875, 0.25537109375, -1.4970703125, -0.7861328125, 1.40625, -2.4140625, 0.98681640625, 1.4150390625, -0.93701171875, -1.4423828125, 0.059051513671875, -0.8974609375, 0.9130859375, -1.05859375, -3.375, -0.54736328125, 0.096435546875, -0.8095703125, -2.068359375, -0.0577392578125, 0.49658203125, -1.4296875, 0.11474609375, 0.392822265625, -0.11981201171875, -0.27734375, 0.2880859375, -0.27294921875, -1.5048828125, -0.361572265625, -1.314453125, -0.285400390625, 0.81494140625, 1.31640625, 0.71240234375, 0.1845703125, 0.75244140625, -0.242919921875, -0.00411224365234375, 0.26171875, -0.78076171875, -1.9921875, -0.1724853515625, 0.0117950439453125, -0.2939453125, -0.42724609375, 0.15966796875, -0.146240234375, 0.98291015625, -0.564453125, 1.818359375, -0.382568359375, 0.032928466796875, 0.51904296875, 0.53369140625, -0.3583984375, -0.712890625, -1.9033203125, 0.353515625, 0.626953125, -0.6845703125, 0.203125, 0.7158203125, -1.9248046875, -0.7509765625, 1.056640625, -1.0205078125, -1.05859375, -0.20654296875, -0.283203125, 1.7138671875, -1.4052734375, -1.8525390625, 0.193603515625, 0.73486328125, 0.7763671875, -0.269287109375, 0.394775390625, -0.2498779296875, 0.55517578125, -0.470703125, 0.400390625, 1.2763671875, -1.6875, -0.580078125, 0.150146484375, -0.95068359375, 1.150390625, -0.9775390625, 0.495849609375, -0.7890625, 1.5576171875, 0.45361328125, -1.900390625, -0.10693359375, -0.268798828125, -1.9228515625, 0.318115234375, 1.935546875, 1.12109375, 1.33984375, 0.904296875, -0.642578125, 0.440185546875, -1.08984375, 0.78955078125, -1.2919921875, 0.5478515625, 1.9794921875, -0.677734375, 0.1585693359375, 0.30029296875, -0.2481689453125, -0.7353515625, -1.57421875, 0.1531982421875, -0.7783203125, 0.26171875, -1.982421875, 2.46484375, -0.65673828125, 0.5703125, 1.3388671875, -0.363525390625, -1.544921875, 1.0546875, 1.1630859375, 3.271484375, -0.51611328125, 2.041015625, -0.456298828125, 0.221923828125, 0.78662109375, -0.276611328125, -0.322021484375, 1.0888671875, 1.7744140625, 0.68896484375, -0.57080078125, -0.366455078125, -0.39599609375, -0.72314453125, 1.130859375, -0.14990234375, -0.658203125, -0.029449462890625, -0.285888671875, -0.9052734375, -1.568359375, -1.3037109375, 0.87158203125, 0.5263671875, -1.33203125, -1.048828125, -0.2276611328125, 0.27880859375, -1.25390625, 0.165771484375, 1.6298828125, -0.53076171875, 0.336181640625, 0.95654296875, 0.418701171875, 0.8564453125, -1.712890625, 0.047027587890625, -1.232421875, 0.1380615234375, -0.85546875, 0.22705078125, 0.63427734375, 0.7705078125, 0.156005859375, -0.7490234375, -0.001880645751953125, -0.256591796875, -0.54931640625, 1.001953125, -0.67236328125, -0.47998046875, -0.3984375, 2.0625, 0.03204345703125, -2.1953125, 0.6142578125, -0.294677734375, -1.7626953125, -1.904296875, 1.234375, -0.193359375, 1.6806640625, -0.56005859375, -0.59228515625, 0.279541015625, 0.365234375, 0.40576171875, 0.9833984375, 0.09844970703125, -0.42236328125, -1.7255859375, 0.225341796875, 1.12890625, 0.943359375, -0.257568359375, -2.05078125, -0.42431640625, 0.6005859375, 0.30712890625, 0.447998046875, -0.422119140625, -0.728515625, 0.322998046875, -0.0341796875, 0.7421875, 1.2890625, -0.830078125, 1.7919921875, -0.572265625, 0.163818359375, -1.421875, -1.28515625, 0.7509765625, 0.1998291015625, 1.58984375, 0.362060546875, -1.146484375, 0.4013671875, -0.83349609375, 0.2479248046875, 1.7109375, 1.9970703125, -0.1153564453125, -1.818359375, 0.10955810546875, 0.352294921875, -0.642578125, -0.44775390625, 0.2421875, 1.318359375, -0.033660888671875, -1.1279296875, 0.7060546875, 0.5625, 0.5341796875, -2.56640625, -0.7265625, -0.264404296875, 3.4375, 1.1708984375, -0.497314453125, -0.53369140625, -0.65380859375, 1.2861328125, -0.81298828125, -0.39990234375, 1.6474609375, 0.305908203125, -1.5927734375, 0.51708984375, 1.505859375, -0.41650390625, 0.86865234375, -0.350341796875, -0.33203125, -0.8349609375, -0.27685546875, -1.765625, 0.252197265625, 1.322265625, -0.7060546875, -0.79638671875, 0.82666015625, -1.2265625, 0.486083984375, -0.54248046875, -0.240966796875, -0.5439453125, -1.4677734375, 0.67529296875, -0.0831298828125, -0.020538330078125, -0.326416015625, 0.2071533203125, -0.77587890625, 0.66845703125, 0.43701171875, 0.1270751953125, 0.205810546875, 0.70703125, -0.0687255859375, -0.354736328125, 1.0390625, 2.693359375, 1.0576171875, 0.31591796875, -1.873046875, 0.462890625, 0.6318359375, -1.3857421875, -0.2132568359375, -1.216796875, 0.306640625, -0.344970703125, -2.1015625, -1.2197265625, 1.15625, -0.304931640625, 0.1175537109375, 1.3408203125, 1.8759765625, 0.529296875, 0.62158203125, -0.1954345703125, 0.451904296875, -1.3837890625, 0.0259246826171875, -0.7421875, 1.1552734375, 0.435791015625, -1.408203125, 0.7333984375, 0.397216796875, -1.11328125, 0.494873046875, 0.310546875, 1.900390625, -0.9306640625, -0.5517578125, 0.07257080078125, 1.4580078125, 0.806640625, 0.54150390625, -0.379638671875, 1.1962890625, -0.57470703125, 1.07421875, 2.671875, 0.346435546875, 0.414306640625, -0.1944580078125, -0.5078125, -0.68115234375, 1.583984375, -1.0791015625, 0.51513671875, -0.74560546875, 0.34228515625, 0.18017578125, 0.341796875, 0.09454345703125, -0.26904296875, -1.287109375, 1.9462890625, -0.56298828125, 1.751953125, 0.2095947265625, 0.78662109375, -0.1610107421875, -3.0, 0.2388916015625, -0.828125, -0.99365234375, 0.51708984375, 0.501953125, -2.5546875, -1.251953125, -0.517578125, 0.00914764404296875, -0.2880859375, 0.05853271484375, 1.5478515625, -0.041961669921875, 0.57421875, -0.671875, -1.53515625, 0.400390625, 0.482177734375, -0.8779296875, -1.1962890625, -0.9521484375, 0.79931640625, -0.371826171875, 0.78955078125, 0.56689453125, -2.12890625, 0.347900390625, 0.16064453125, 0.1317138671875, -0.378662109375, -1.1416015625, -0.1705322265625, -0.568359375, -1.234375, -0.3896484375, 1.361328125, 0.42724609375, 1.63671875, -2.1328125, -0.052398681640625, -0.43115234375, -0.40234375, -0.89111328125, 0.030731201171875, 0.4130859375, -0.143310546875, 0.6650390625, -0.4140625, -0.1873779296875, 0.421142578125, -0.60205078125, -1.2607421875, -2.681640625, -0.177734375, -0.470458984375, -0.01026153564453125, 1.0576171875, 0.60009765625, -0.28173828125, -0.7060546875, 0.759765625, -0.6142578125, 2.14453125, -0.0012388229370117188, -1.013671875, -0.4765625, -1.291015625, 1.33984375, -0.671875, -1.30859375, 2.556640625, 0.241455078125, -1.2451171875, -0.0280303955078125, -1.109375, -0.409423828125, -0.7138671875, -0.060455322265625, -0.978515625, -0.35693359375, -0.646484375, 0.283203125, -0.045166015625, -1.294921875, -0.428955078125, 0.443359375, 0.146484375, 0.06353759765625, 1.9453125, 0.450439453125, 0.270263671875, -0.0095672607421875, 1.1435546875, 0.353271484375, 0.155029296875, 1.1171875, -1.5380859375, 0.15625, -1.15234375, 1.1884765625, 0.6923828125, 0.31591796875, 0.283203125, 1.7861328125, -0.486083984375, 0.611328125, 1.845703125, -0.51123046875, -0.279052734375, -0.7021484375, 0.5771484375, 0.436279296875, -1.6875, -0.338623046875, -0.08673095703125, 0.83349609375, -1.64453125, 0.57666015625, 0.44189453125, 0.11181640625, 1.3193359375, 0.52490234375, 0.63671875, -0.442138671875, 1.29296875, -0.3271484375, -1.1640625, 2.26171875, 1.15234375, -0.1568603515625, 0.994140625, 0.81640625, 1.9111328125, -0.64306640625, 1.28125, 0.383056640625, 0.69921875, 0.97607421875, 0.479248046875, -0.57470703125, 0.73486328125, -1.3974609375, 1.2021484375, 0.265625, -0.184814453125, -0.76611328125, -1.0087890625, -0.7802734375, -0.3232421875, -2.03125, 0.11151123046875, 0.453369140625, -1.244140625, 0.7861328125, 0.331787109375, -0.89404296875, 0.52197265625, 0.1490478515625, 0.425537109375, 0.3720703125, -1.1943359375, -0.19482421875, -0.84423828125, -1.6875, 0.048828125, 0.669921875, 0.384033203125, 1.9765625, -0.0859375, -0.849609375, 0.2020263671875, 0.77734375, 0.46875, -0.77197265625, -1.3046875, -1.1767578125, 0.059051513671875, 1.1552734375, -0.5380859375, 0.66552734375, -0.398193359375, -1.6259765625, -0.482177734375, 0.468017578125, -0.168701171875, -0.65234375, -0.031707763671875, 1.2431640625, 0.95654296875, 1.400390625, -1.1328125, -0.70703125, 1.1806640625, -0.49169921875, 0.53125, -1.4755859375, -0.8271484375, 0.168701171875, 0.435546875, -1.1435546875, -1.2626953125, -1.25, -0.2242431640625, 0.09820556640625, -0.57666015625, 0.51708984375, 1.5625, -0.9013671875, -0.32470703125, 0.98681640625, 1.330078125, 0.8349609375, 0.051025390625, 1.1826171875, -0.4189453125, 0.22265625, 0.2330322265625, 1.640625, 1.1533203125, -0.7275390625, -0.443603515625, -1.4931640625, -0.263671875, -1.171875, 2.17578125, 0.388671875, -0.7060546875, 0.8564453125, 0.27685546875, -0.9677734375, -2.169921875, 0.71630859375, 0.377197265625, -0.9423828125, 0.97216796875, 0.232666015625, 0.3623046875, 1.9580078125, -1.388671875, 0.494873046875, 0.53466796875, -0.6201171875, -0.96630859375, 0.97021484375, 0.37353515625, -1.447265625, 0.283935546875, -0.94482421875, -1.400390625, 0.8642578125, 0.34326171875, 0.57275390625, 1.0576171875, -1.5068359375, -0.9853515625, -0.03900146484375, -0.64794921875, -0.9423828125, 0.62841796875, 2.341796875, 0.333251953125, 1.4345703125, -0.27197265625, -0.5478515625, 2.529296875, 0.11474609375, -0.310546875, -1.3310546875, -0.56005859375, 0.65087890625, 0.81689453125, 1.087890625, -1.40625, -2.802734375, 1.4736328125, 0.5498046875, -1.056640625, 0.7099609375, -0.260009765625, -1.419921875, -1.482421875, 0.81689453125, -0.7802734375, 1.7021484375, 0.0048828125, 0.94677734375, -0.6708984375, -0.1815185546875, 1.3037109375, -0.53369140625, 0.28125, -0.39208984375, 0.55810546875, -0.93505859375, 0.11187744140625, -0.11090087890625, -0.052459716796875, -0.2069091796875, -1.375, -0.23486328125, -0.310302734375, -0.1759033203125, 0.76611328125, -0.46044921875, -0.303466796875, 2.150390625, 0.02484130859375, 0.06439208984375, 1.4970703125, 0.4111328125, 0.6474609375, 1.5107421875, -0.64501953125, -0.053314208984375, -0.56298828125, -1.625, -2.314453125, 0.0303192138671875, 0.0083770751953125, 0.9462890625, -0.051788330078125, 0.51904296875, 0.60205078125, -0.7900390625, 0.0082550048828125, 0.031829833984375, -1.5947265625, 0.415771484375, -0.271484375, -1.23828125, 1.087890625, -0.60595703125, -0.151123046875, -1.25, -0.9853515625, 0.484619140625, 0.9775390625, 1.685546875, 0.9853515625, 0.43408203125, -1.365234375, -0.3916015625, 0.2186279296875, 0.70361328125, -0.7412109375, 1.138671875, 0.5068359375, 1.5537109375, -0.73046875, 0.19921875, -0.360107421875, -1.7919921875, 0.1405029296875, -0.7001953125, -0.233642578125, 0.456298828125, -0.400634765625, -1.037109375, 0.419677734375, -0.79052734375, 0.1533203125, 1.029296875, -0.346923828125, -0.51953125, -0.153564453125, -1.3916015625, 2.048828125, 0.52587890625, 0.06292724609375, -0.74951171875, 1.5107421875, 0.003620147705078125, 1.734375, 0.5595703125, -1.30859375, 0.432861328125, -0.0305023193359375, -0.434814453125, -0.71826171875, -0.402587890625, -0.0931396484375, 0.96240234375, -0.0102386474609375, -1.572265625, -0.51611328125, 0.60498046875, 1.9638671875, 0.5185546875, 0.818359375, 0.280029296875, 0.44921875, -0.6376953125, -0.4638671875, 0.192138671875, 0.348388671875, -0.9091796875, -1.6337890625, 0.3173828125, -0.59423828125, -1.142578125, 1.0185546875, -1.5361328125, -0.40576171875, 0.900390625, 1.0986328125, 0.5732421875, 0.005352020263671875, 0.05450439453125, -0.1639404296875, 0.371337890625, -1.3388671875, -0.458740234375, 1.3701171875, 1.564453125, -0.2144775390625, 0.268310546875, 0.1826171875, -1.0205078125, -0.47412109375, 1.677734375, -0.09027099609375, 0.383056640625, -0.340576171875, 0.5283203125, 2.1796875, 1.7294921875, 0.927734375, 0.8251953125, -2.259765625, -0.0672607421875, 1.4130859375, -1.6083984375, -0.245361328125, 0.3896484375, 1.9970703125, 0.91748046875, -0.281494140625, -1.3154296875, -0.80322265625, 1.7724609375, 1.2158203125, 2.015625, 1.0849609375, 2.376953125, -0.00701904296875, -0.1500244140625, -1.673828125, 0.376953125, -0.385986328125, 1.0625, 0.465576171875, -1.1455078125, 0.71142578125, -0.751953125, 1.0224609375, 0.4755859375, 0.470458984375, 0.7685546875, 0.23486328125, 0.126953125, 2.080078125, 1.0341796875, 0.134521484375, -0.129638671875, -3.150390625, 0.78515625, 0.998046875, 1.3046875, -1.1083984375, -0.82177734375, -0.391845703125, 0.09722900390625, 0.6083984375, -0.2130126953125, -0.443359375, -0.401123046875, -0.428466796875, 0.82568359375, 0.8310546875, -0.93017578125, 0.0655517578125, -1.2080078125, -0.6298828125, 0.342041015625, 0.919921875, -0.751953125, 0.373046875, 0.6083984375, 0.1263427734375, -2.685546875, -0.8486328125, -1.435546875, -1.1298828125, 1.7490234375, 1.2431640625, 0.63134765625, 1.49609375, 0.2459716796875, -1.0546875, -0.11431884765625, 1.5263671875, -1.08984375, -1.513671875, -0.7021484375, -0.332763671875, 0.56787109375, 0.4267578125, -0.64990234375, 0.29736328125, -1.2021484375, -2.6171875, 0.5302734375, -0.5009765625, 0.487548828125, -0.72509765625, -0.151123046875, 0.6796875, -1.9912109375, -0.5068359375, -1.44921875, -0.4150390625, 0.623046875, -0.85302734375, -0.2239990234375, -0.3486328125, 0.58251953125, -0.389892578125, 0.38818359375, 1.0185546875, -0.30517578125, 0.5205078125, -1.130859375, -0.91357421875, -0.91552734375, -0.2279052734375, 1.9912109375, -0.3251953125, -1.509765625, -0.91552734375, 0.1002197265625, 0.650390625, -0.330078125, 0.9521484375, -0.60205078125, -0.485107421875, 0.317626953125, -2.34375, -0.671875, -0.09112548828125, -0.2587890625, -0.88427734375, 1.0625, 1.42578125, -1.2607421875, -0.60986328125, 0.63671875, 1.916015625, -0.60888671875, -0.0117034912109375, -0.994140625, 0.83935546875, -0.1619873046875, 1.67578125, 1.2861328125, -1.0927734375, 1.1337890625, 1.83984375, 0.5126953125, 0.7470703125, -0.6416015625, -1.38671875, -0.1807861328125, -1.486328125, -2.37890625, 0.9921875, 0.5029296875, -2.048828125, -0.931640625, 2.076171875, -0.279541015625, -1.3076171875, -0.65966796875, 1.0126953125, 1.501953125, -0.6259765625, 0.09454345703125, 1.66796875, 1.01171875, -1.072265625, 0.130615234375, -0.345703125, 1.6044921875, 1.2705078125, 0.7490234375, 1.3232421875, 0.400146484375, 0.1776123046875, -0.135009765625, 0.78759765625, 0.49267578125, 0.60302734375, -0.337646484375, -0.311767578125, 1.8671875, 2.0546875, 0.93603515625, 1.0185546875, 0.7080078125, -1.6171875, -0.70654296875, -0.401123046875, -0.30029296875, 0.340087890625, -0.9345703125, 0.93115234375, -0.0140838623046875, -0.042572021484375, -0.1068115234375, 0.86669921875, -0.5810546875, 0.435546875, -0.92578125, 0.367919921875, 1.1318359375, 1.15234375, 1.4814453125, -0.043731689453125, 0.310546875, -1.0400390625, -0.5283203125, -0.91650390625, 0.382568359375, 1.212890625, -0.548828125, 1.0556640625, 0.1942138671875, 2.2890625, -0.90380859375, 1.587890625, -0.002841949462890625, -0.72412109375, 0.7236328125, -0.381591796875, -1.1640625, 1.6708984375, -0.311279296875, -0.12127685546875, 0.465576171875, 0.323486328125, 0.47216796875, -0.5390625, 0.45947265625, -0.62109375, -0.67919921875, -0.046417236328125, 1.0654296875, 0.302734375, -0.5478515625, -0.12646484375, 0.12261962890625, 1.2158203125, 0.27685546875, -0.0902099609375, -0.1514892578125, 0.03192138671875, -0.4599609375, 0.5263671875, -0.2320556640625, 1.244140625, -0.9462890625, 1.12890625, -0.2392578125, 1.7607421875, 0.67578125, -1.1201171875, -1.466796875, -1.0908203125, 1.1298828125, -0.20947265625, -1.0302734375, 2.201171875, 1.1630859375, 1.07421875, 0.0531005859375, -0.87744140625, 0.8447265625, 0.74365234375, -0.6181640625, -0.603515625, -0.0970458984375, -1.142578125, 0.339111328125, 0.5673828125, -0.177978515625, 0.0225982666015625, -0.81591796875, -0.71044921875, -0.04296875, -1.5068359375, 1.109375, -0.332275390625, 0.03192138671875, -0.1884765625, 1.12890625, 0.42431640625, -0.9833984375, -0.1756591796875, -2.376953125, -0.52880859375, 1.115234375, -0.63916015625, 1.6865234375, -0.591796875, -1.568359375, -2.33203125, -1.2763671875, 0.162109375, -1.58203125, 0.9521484375, -0.2283935546875, 0.83837890625, 0.93408203125, -0.435791015625, -0.26416015625, 0.638671875, -1.2119140625, -0.94580078125, -1.0771484375, 0.8095703125, 1.73828125, -1.275390625, 0.470458984375, 1.650390625, 0.947265625, 0.1734619140625, 1.8056640625, -0.49267578125, -0.1942138671875, -0.3134765625, -2.9609375, -0.5634765625, -0.2318115234375, -0.79736328125, 0.5693359375, -0.08355712890625, -1.0556640625, 0.1514892578125, -0.9384765625, -0.626953125, -0.73583984375, 0.763671875, -1.287109375, 0.521484375, -1.1025390625, 0.96533203125, -1.3544921875, 0.056243896484375, -0.4755859375, 0.91357421875, -0.299072265625, 0.60302734375, -1.23046875, -0.87353515625, 0.81787109375, -0.268310546875, -0.62255859375, 0.11944580078125, 0.1851806640625, -0.016357421875, 1.1103515625, -0.80419921875, -1.1953125, -0.328857421875, 0.144775390625, 0.26953125, -0.3046875, 0.287841796875, -0.55322265625, -0.396240234375, 0.2435302734375, 0.1656494140625, -1.5849609375, -0.2080078125, 0.5927734375, -0.56787109375, -1.595703125, -1.068359375, 0.001430511474609375, 0.02557373046875, -0.270751953125, -1.3671875, -0.1739501953125, -0.95263671875, 1.2255859375, -0.377685546875, -2.216796875, 0.7939453125, -0.059722900390625, 0.84912109375, -0.701171875, 1.0732421875, -1.6044921875, 1.3681640625, -0.58056640625, 1.09375, -0.2451171875, -0.0687255859375, 0.451904296875, 0.76416015625, -0.63916015625, -1.416015625, 0.12042236328125, -0.66162109375, 1.16796875, -0.8974609375, -0.634765625, 0.87255859375, -1.2060546875, -0.489501953125, 0.385498046875, -1.341796875, 1.1298828125, 0.896484375, -1.83984375, 2.1015625, -0.5927734375, -0.72900390625, -1.4736328125, -1.1318359375, -0.19091796875, -1.3046875, 0.038818359375, -1.3623046875, -0.71630859375, 1.2763671875, 0.7275390625, 0.66455078125, 1.3212890625, 0.479736328125, -2.56640625, 0.314453125, 0.09393310546875, -0.1060791015625, 1.6240234375, -0.261962890625, -0.1435546875, -0.0804443359375, -0.491943359375, -1.94140625, 0.06488037109375, 0.1131591796875, 0.74609375, -0.9208984375, 0.783203125, -0.99169921875, 2.171875, -1.390625, 0.35693359375, -0.29931640625, 0.050811767578125, -0.798828125, 1.4443359375, 0.5048828125, 1.8544921875, -1.0390625, -0.6728515625, 1.384765625, -1.3857421875, -0.25390625, 0.3642578125, -1.87890625, -1.0615234375, 0.537109375, -0.418212890625, 0.0328369140625, -0.25439453125, 0.93408203125, 0.21875, -1.052734375, -0.4765625, -0.5400390625, 0.0294342041015625, 0.85498046875, -0.96826171875, 1.2197265625, -0.278564453125, 1.4873046875, 0.65185546875, 0.50537109375, -0.11187744140625, -1.048828125, 1.6630859375, -1.203125, 1.326171875, -3.650390625, -0.2039794921875, -0.025909423828125, 1.2412109375, 1.1513671875, -0.6552734375, 1.0478515625, 0.3193359375, -0.408203125, -2.2421875, 0.19482421875, 0.73583984375, -0.06689453125, -2.00390625, -0.4912109375, -1.654296875, 0.2265625, -1.248046875, -1.6650390625, 0.426513671875, 0.05108642578125, 0.311279296875, 1.70703125, 0.52685546875, 1.3798828125, -2.314453125, -0.38525390625, -0.0858154296875, 1.44140625, 0.59521484375, 3.26171875, -0.0214385986328125, -1.9482421875, -1.2158203125, 2.18359375, -1.208984375, 0.63720703125, 0.6015625, 0.63525390625, 1.18359375, 1.1796875, -0.000232696533203125, 0.7255859375, -0.1982421875, -0.2352294921875, 0.8291015625, -0.18359375, 0.91162109375, 0.9521484375, -0.2264404296875, 0.494873046875, 0.250244140625, 0.7060546875, -1.2451171875, 1.248046875, 0.300537109375, 0.56640625, -0.6845703125, 0.94189453125, 1.4248046875, -1.8974609375, -0.47705078125, -0.30322265625, 0.2666015625, -0.96875, -2.046875, 1.310546875, 0.182861328125, 0.488525390625, 0.85693359375, -1.55859375, 0.037078857421875, -1.7216796875, 1.9345703125, 0.2130126953125, -0.65673828125, -0.292724609375, -0.239990234375, 0.58837890625, -0.8720703125, 0.02642822265625, 0.92529296875, 0.1895751953125, -0.264892578125, -0.59130859375, 0.10040283203125, -1.0380859375, -0.29638671875, -0.904296875, -0.455322265625, 0.591796875, 0.1470947265625, 0.7451171875, -1.1064453125, -0.2293701171875, -0.68603515625, 1.8740234375, -0.292236328125, 0.826171875, 0.56396484375, -1.169921875, -2.23828125, -0.02777099609375, -0.0499267578125, -0.72998046875, 0.91015625, 0.2266845703125, 1.650390625, -0.035003662109375, 2.359375, -0.2705078125, -0.83056640625, -0.302490234375, 0.030975341796875, -0.556640625, -0.9326171875, -0.2330322265625, -0.28369140625, 1.576171875, -0.939453125, 0.94482421875, 1.2255859375, -0.40380859375, -0.265380859375, -0.275634765625, -0.371826171875, -1.0625, 0.9609375, -0.43896484375, 0.73388671875, 2.072265625, -0.1927490234375, 0.814453125, 0.35107421875, -0.037567138671875, -0.20703125, -1.2177734375, -0.2493896484375, -0.89794921875, 0.04693603515625, -0.7001953125, 0.201904296875, 0.25244140625, -0.740234375, 0.5810546875, -0.90185546875, 1.2890625, 0.4365234375, -0.28515625, -1.2548828125, -1.435546875, -1.2900390625, -0.350830078125, -0.84326171875, -0.428466796875, 1.126953125, 1.10546875, -1.3984375, 0.50439453125, -0.52880859375, 0.65869140625, -0.313232421875, -0.0297088623046875, -0.2254638671875, 1.1943359375, -1.32421875, -0.978515625, 0.92919921875, -0.011505126953125, -0.059722900390625, 0.1595458984375, 0.1834716796875, -1.890625, 1.18359375, 0.7236328125, -0.84716796875, -0.93603515625, 0.1844482421875, 0.6552734375, -0.468994140625, 0.4951171875, -1.119140625, -0.27294921875, 2.890625, -0.10491943359375, 0.5927734375, 0.5009765625, -0.55810546875, -0.70703125, 0.429931640625, 1.6787109375, 1.9482421875, 1.0146484375, 0.453857421875, -0.568359375, -3.083984375, -2.36328125, 2.400390625, 1.236328125, -0.8837890625, -0.01107025146484375, 0.85791015625, 0.250732421875, 1.3056640625, 0.85302734375, -0.10894775390625, -0.404052734375, 0.421630859375, -0.32177734375, -0.7109375, -0.26953125, 0.485595703125, 1.09375, 0.2322998046875, -0.11962890625, 1.2939453125, 0.9013671875, 0.1558837890625, -0.1807861328125, -0.58447265625, -0.60400390625, 0.058258056640625, -0.03363037109375, 1.796875, 1.5224609375, -0.0511474609375, -0.345947265625, -0.48193359375, -1.1796875, -0.9541015625, -0.42919921875, -0.37890625, -0.8173828125, 0.0138397216796875, -0.6064453125, -1.8310546875, -0.99951171875, -0.478271484375, -0.06781005859375, -0.470947265625, 1.4130859375, 0.2208251953125, -0.92431640625, -0.40234375, -1.1005859375, 2.236328125, -1.263671875, -2.2109375, 0.361328125, -0.2423095703125, 1.876953125, 0.405517578125, 0.7666015625, 0.215087890625, 2.56640625, 0.442626953125, 0.1478271484375, 0.85693359375, 1.046875, -0.450439453125, 0.1456298828125, 1.234375, -1.8271484375, -1.51953125, -0.5205078125, 1.85546875, -0.0660400390625, 1.3115234375, 1.96875, -0.5693359375, 0.164306640625, 0.72314453125, -1.91796875, -0.69970703125, 1.9140625, -1.4716796875, 0.6650390625, 0.5888671875, 0.422119140625, 0.032562255859375, -1.333984375, -0.947265625, -0.417236328125, -1.287109375, 0.08966064453125, 0.415283203125, -1.1416015625, -0.6533203125, 0.92236328125, 0.52197265625, 0.1805419921875, -0.150146484375, -1.478515625, 1.2431640625, -0.794921875, 0.65478515625, -0.2017822265625, -0.0823974609375, -1.33984375, -0.10723876953125, 0.79345703125, -0.0088348388671875, -1.001953125, 0.140625, 0.52783203125, -1.171875, 0.428466796875, 0.4326171875, 1.451171875, -0.28466796875, -0.07666015625, 0.427978515625, 1.388671875, -0.68896484375, 1.185546875, -0.47119140625, 0.560546875, -1.154296875, 1.912109375, -1.1181640625, -1.2490234375, 0.9033203125, -1.849609375, 0.81640625, -0.5771484375, 2.212890625, -0.269287109375, -0.5810546875, -0.410400390625, -1.771484375, 0.2000732421875, 1.888671875, 0.34130859375, -1.8408203125, 0.2457275390625, -1.19140625, -0.480712890625, -0.6083984375, 1.0859375, 0.184814453125, -1.228515625, -1.91796875, -1.322265625, 0.11798095703125, 1.6083984375, 0.8720703125, -0.07867431640625, 0.55126953125, -1.4091796875, -0.1561279296875, 0.42626953125, 1.1005859375, -0.15869140625, 0.8994140625, -1.0458984375, -0.7421875, 0.06353759765625, 0.78564453125, -0.7177734375, -0.59326171875, -1.216796875, 0.693359375, -0.424072265625, 1.1552734375, -0.595703125, -1.9296875, -0.7890625, -2.05078125, -1.0478515625, 0.1768798828125, -0.10760498046875, -0.78759765625, 0.279541015625, -0.1436767578125, 0.035186767578125, 0.72021484375, 1.517578125, -0.69970703125, 1.5419921875, -1.201171875, -0.3095703125, -0.67431640625, -0.2293701171875, -1.435546875, 1.451171875, -0.27685546875, 0.916015625, 0.67919921875, 0.7333984375, 0.72509765625, 0.83203125, -0.11328125, 1.0322265625, 0.96875, -0.64599609375, 1.21875, -0.54345703125, -1.1865234375, 1.9072265625, -1.3486328125, 1.2216796875, 1.958984375, 0.1800537109375, 0.72119140625, -0.355224609375, -0.75927734375, 0.9775390625, 1.552734375, 0.3037109375, 0.185791015625, 0.67138671875, -2.5, 1.2412109375, -1.06640625, -1.2236328125, 0.9921875, -0.7255859375, -0.720703125, -1.02734375, 0.90625, 0.8818359375, 0.91552734375, 1.134765625, 0.286376953125, 1.4775390625, -0.470947265625, -0.11895751953125, -0.391357421875, -1.494140625, 1.123046875, 1.83203125, -1.234375, -0.145263671875, 0.395263671875, 0.2958984375, 0.7314453125, -0.50830078125, -0.556640625, 0.372802734375, 0.315673828125, 1.5615234375, 1.6005859375, 1.302734375, 0.77978515625, 0.98486328125, 0.491455078125, 0.77734375, 0.9892578125, -0.482666015625, -1.0576171875, -0.499755859375, -0.85888671875, 1.892578125, -0.234619140625, -0.265380859375, -0.5859375, 0.81396484375, -0.54052734375, 0.1781005859375, -0.205322265625, -0.274658203125, -0.0093994140625, -1.6171875, -1.705078125, -0.37939453125, 0.0609130859375, -0.66455078125, -0.164306640625, -0.857421875, 0.501953125, 0.46630859375, 1.837890625, -1.708984375, -0.71533203125, 0.77783203125, -0.162109375, -0.70068359375, 0.95068359375, -0.21142578125, -1.1572265625, 1.10546875, -0.8759765625, 0.112548828125, -1.2177734375, -0.72021484375, -1.921875, -0.58447265625, -0.341552734375, -0.1884765625, -1.0048828125, 0.6923828125, 0.471923828125, 2.0703125, -2.138671875, -1.052734375, -0.55078125, -0.67919921875, -0.23876953125, -0.68603515625, 0.3212890625, 0.2352294921875, -0.442626953125, -0.07635498046875, 0.62060546875, 2.1796875, -2.029296875, 0.353515625, -0.096923828125, 0.1424560546875, -0.77783203125, 0.424560546875, 2.22265625, -0.56689453125, -1.4580078125, -0.59912109375, 0.1297607421875, 1.55078125, -0.8896484375, 1.2509765625, -0.350830078125, -0.68994140625, 1.404296875, 0.3701171875, 1.6416015625, -2.61328125, -0.9453125, 0.496337890625, -0.552734375, -0.64111328125, 0.11785888671875, 0.923828125, -0.4677734375, -0.55029296875, 0.1728515625, -1.015625, -1.740234375, 0.69140625, -0.67236328125, 0.912109375, 1.0615234375, 0.62158203125, 0.0465087890625, 0.89990234375, -0.47900390625, 2.275390625, 0.2418212890625, 0.79345703125, -0.474853515625, 0.81298828125, 1.755859375, -0.6337890625, 0.5419921875, 0.387451171875, -0.314697265625, -0.98046875, 1.0087890625, -0.734375, -0.580078125, 0.14111328125, -0.69140625, 0.61865234375, -0.62646484375, -0.75537109375, 0.306640625, -0.442626953125, -0.253173828125, -0.7685546875, -0.7275390625, -0.9482421875, 0.365234375, 0.8896484375, -0.3916015625, -0.6435546875, 1.400390625, 0.74951171875, 0.9677734375, -1.0478515625, -0.21826171875, -0.270751953125, 0.07769775390625, -0.875, -0.390380859375, -2.103515625, -1.8203125, 0.08953857421875, -0.66455078125, 0.8984375, 0.14208984375, 0.440185546875, 0.1434326171875, 0.69921875, -0.85595703125, -1.017578125, -0.66845703125, -0.29150390625, -0.332275390625, 1.080078125, -0.286376953125, 0.73291015625, 0.06463623046875, -0.6455078125, -0.364990234375, 0.98046875, 0.03204345703125, -0.435791015625, -0.76708984375, -1.908203125, 0.10125732421875, 0.0714111328125, 0.16259765625, 1.0400390625, -1.419921875, -0.1796875, 0.671875, 1.517578125, 2.1875, -0.2371826171875, -0.024658203125, -0.6298828125, 0.61767578125, 0.7177734375, -2.57421875, 1.068359375, 0.60205078125, -0.3974609375, 0.31591796875, 1.4560546875, -0.7509765625, -1.701171875, -0.50732421875, -0.0079345703125, 0.8544921875, -0.107421875, -0.281005859375, -0.654296875, 1.064453125, -0.427490234375, -0.98486328125, -0.7568359375, 0.339111328125, 0.9326171875, -1.14453125, 1.767578125, 0.312255859375, -2.0078125, 0.89306640625, 0.705078125, -1.353515625, -0.63427734375, 0.57177734375, -0.939453125, -1.365234375, 0.250244140625, 0.26318359375, -1.0166015625, 0.2298583984375, 0.7861328125, 0.71044921875, -0.75927734375, -1.134765625, -0.6025390625, 0.53369140625, 1.3779296875, -0.474365234375, 1.400390625, 0.6171875, 0.755859375, 0.79443359375, 0.984375, -0.325927734375, 0.67529296875, -0.6669921875, 0.371826171875, -0.31591796875, 1.375, 0.1634521484375, 0.376953125, 0.9609375, 0.92333984375, -1.3603515625, -0.884765625, 1.4462890625, 0.08843994140625, -0.572265625, -0.3212890625, -0.166259765625, 0.321044921875, -0.8310546875, 0.67333984375, 0.39697265625, -0.3671875, -1.384765625, 0.11590576171875, 0.77587890625, 0.6953125, 0.25634765625, 0.5205078125, 0.35009765625, 0.0157623291015625, -0.1943359375, 0.724609375, -0.2861328125, 0.3583984375, -1.3046875, -1.0107421875, 0.5908203125, -0.482666015625, -1.6953125, 0.52392578125, 1.0087890625, -0.1263427734375, 0.65087890625, 0.087890625, 0.100830078125, -1.9892578125, -0.73876953125, -0.42138671875, -2.5234375, 0.01181793212890625, -0.34423828125, -0.419677734375, 2.93359375, 1.3623046875, 0.67138671875, -0.1873779296875, 0.430908203125, 0.72021484375, -0.759765625, 0.08343505859375, -0.52978515625, -0.68505859375, 0.8818359375, -0.90576171875, 1.7548828125, 0.2484130859375, 0.50732421875, -0.0046234130859375, -0.47802734375, 0.6923828125, 0.62255859375, -0.5546875, 1.044921875, -0.20654296875, -0.60302734375, 0.1456298828125, -0.974609375, 0.415771484375, 0.216796875, -0.4619140625, 1.388671875, 0.46923828125, 0.97900390625, 0.875, -0.31884765625, 0.81640625, 0.8427734375, 0.0865478515625, 1.484375, 0.326904296875, -1.248046875, 0.170654296875, 0.393310546875, 1.4384765625, 1.1083984375, 0.53125, 1.19140625, -0.26416015625, -2.109375, -0.4755859375, -2.0390625, -1.30078125, -1.8310546875, 1.3134765625, -0.4765625, -0.35498046875, 0.0268707275390625, -0.81396484375, -0.1505126953125, -0.5244140625, -0.96533203125, 0.463134765625, -0.6875, 1.8017578125, -0.1729736328125, 0.471435546875, -0.1312255859375, -0.2283935546875, 0.032379150390625, 1.3056640625, 0.00315093994140625, 1.1103515625, -0.250732421875, -0.07269287109375, 1.359375, 0.50927734375, -1.6826171875, -0.88427734375, 0.53173828125, -0.80029296875, 0.2303466796875, 0.51513671875, 1.84375, -0.6279296875, 0.45068359375, -0.28564453125, 0.181396484375, 1.8173828125, -0.17822265625, -0.79150390625, 0.4453125, 0.07373046875, 1.0810546875, 0.75, 0.75537109375, 0.321044921875, 1.3115234375, -1.0576171875, -0.66748046875, -0.029632568359375, -0.250244140625, 1.1474609375, 1.8720703125, -0.32177734375, -0.9248046875, -0.66357421875, -0.91845703125, 0.91015625, -1.3349609375, -0.367919921875, 0.15380859375, 0.1749267578125, -0.70458984375, -0.76904296875, 0.89599609375, 1.2783203125, -1.6279296875, 0.55517578125, 0.2900390625, 1.4189453125, -1.6708984375, -0.255126953125, -0.2939453125, -0.0865478515625, 0.1622314453125, -2.072265625, -0.94482421875, 1.1650390625, 2.94921875, -0.1632080078125, 0.69482421875, -1.6318359375, -0.29833984375, -1.4033203125, 0.4306640625, -0.047332763671875, -1.388671875, -2.669921875, 1.4169921875, 0.230712890625, -0.392333984375, 1.6982421875, 2.125, -0.30859375, -0.376708984375, -1.345703125, -0.40966796875, 0.54736328125, -0.8349609375, -1.05078125, 0.0204010009765625, 0.7685546875, -1.068359375, 0.74560546875, -1.287109375, 0.58349609375, 0.61181640625, 0.99560546875, -0.5029296875, -0.67333984375, -1.1083984375, 0.8779296875, 0.640625, -0.94921875, -0.5029296875, -1.439453125, -0.88623046875, 1.2314453125, -1.59375, 1.46484375, 0.71728515625, 0.8955078125, 1.9384765625, 1.673828125, 0.208740234375, 0.2666015625, -0.1553955078125, -0.3837890625, 0.6943359375, 0.03009033203125, -0.6826171875, -0.40576171875, -0.331298828125, -0.1873779296875, -0.82080078125, 1.74609375, 2.87109375, -0.68408203125, 1.3740234375, 1.3115234375, -1.572265625, -0.2310791015625, -0.67431640625, -0.29931640625, -0.8818359375, 0.488525390625, -1.1982421875, -0.56005859375, 1.0390625, -0.5654296875, -0.55322265625, 0.70458984375, -0.99951171875, 0.42919921875, 0.78125, 0.464599609375, -0.65966796875, 0.265380859375, 2.943359375, -1.4208984375, 0.365478515625, 0.6513671875, 2.017578125, 1.3759765625, 0.63232421875, 0.301025390625, -2.0, -0.86279296875, -0.91796875, -0.36279296875, -1.3369140625, 0.083251953125, -0.315673828125, -0.1397705078125, 0.1806640625, -2.083984375, 0.41455078125, -0.71044921875, 1.0908203125, -0.018096923828125, 0.564453125, 0.2047119140625, -0.039398193359375, 0.78466796875, 0.64208984375, -1.4599609375, -0.90869140625, 0.94580078125, 1.5927734375, -0.599609375, -0.86279296875, 2.2734375, -0.65673828125, -0.08056640625, 0.4677734375, 1.119140625, 0.048004150390625, 1.4716796875, 0.06341552734375, 0.042755126953125, 0.333251953125, -0.92041015625, -2.138671875, -1.0205078125, -0.0259857177734375, 0.3173828125, 1.505859375, 1.05859375, 1.2900390625, 0.164794921875, 0.50390625, -0.91015625, -0.74853515625, -0.56103515625, -0.771484375, 0.92236328125, 0.27880859375, 0.2763671875, -0.81201171875, 0.1798095703125, 0.386962890625, 0.277099609375, 1.478515625, -2.291015625, 1.1767578125, -0.71923828125, -0.42236328125, 0.79541015625, 0.54931640625, 0.65673828125, 0.446044921875, 1.994140625, 0.76904296875, 0.028411865234375, -1.0751953125, 0.42431640625, 0.54296875, -1.435546875, -1.0224609375, -1.734375, 0.10418701171875, -0.114013671875, 1.4287109375, -0.697265625, 0.2138671875, 0.72216796875, -0.08697509765625, -0.93359375, -0.03704833984375, 0.67822265625, -0.1072998046875, -0.6611328125, 0.55029296875, -0.89599609375, -1.0751953125, 0.5400390625, 0.6181640625, 0.104248046875, 0.3056640625, -0.67236328125, -0.5224609375, 1.509765625, 0.53271484375, 0.2327880859375, -0.501953125, 0.5166015625, 0.00946044921875, 1.84375, 0.40625, 0.489501953125, 0.5791015625, 0.215087890625, 0.373291015625, 1.544921875, 0.424072265625, 1.8564453125, 0.67919921875, -0.2408447265625, -0.42041015625, 1.546875, -2.74609375, 0.359619140625, 0.429443359375, -1.501953125, -0.47998046875, -0.486572265625, -1.3017578125, 0.103271484375, -2.634765625, 0.6279296875, -1.9150390625, -2.93359375, 0.9638671875, 0.57666015625, -0.292236328125, -0.59521484375, -1.7490234375, 0.417724609375, 0.25537109375, 0.0933837890625, 0.42138671875, 0.74658203125, 1.42578125, -1.189453125, 0.48046875, 1.03515625, 1.5703125, 0.927734375, 0.9970703125, 0.111083984375, 0.705078125, -1.8388671875, -0.2099609375, -0.4267578125, -1.13671875, 0.199462890625, -1.4228515625, 1.7197265625, -3.140625, -0.355224609375, -0.477783203125, -0.340576171875, 0.0770263671875, 1.125, 0.253173828125, 0.68896484375, 1.1865234375, 0.91748046875, 0.2452392578125, -0.2410888671875, -1.4111328125, -0.440185546875, 1.435546875, 0.2607421875, -0.2802734375, 0.06097412109375, -0.10406494140625, -0.8740234375, 1.271484375, -0.958984375, -1.3251953125, 0.71826171875, 0.01788330078125, 0.77783203125, -1.068359375, 0.9716796875, 0.24267578125, -1.314453125, -0.423583984375, 0.9609375, -0.6201171875, -1.7529296875, -1.869140625, -0.8671875, 0.18603515625, -0.10626220703125, 1.4873046875, -0.97802734375, 0.05682373046875, -1.087890625, -0.37158203125, -0.5732421875, 0.03399658203125, 0.79931640625, -1.39453125, -0.388427734375, -1.0791015625, -0.642578125, -0.69677734375, -0.8779296875, 1.1845703125, -1.77734375, -1.1767578125, -2.0234375, -0.6591796875, 2.115234375, 0.75537109375, 0.8125, 1.1591796875, 0.471923828125, 2.4765625, -1.05859375, -1.6904296875, 1.291015625, -2.12109375, 0.1636962890625, -0.464599609375, -1.689453125, 0.68798828125, -0.274169921875, 0.884765625, 0.147216796875, 0.771484375, 1.458984375, 0.423828125, 0.873046875, -0.36083984375, 0.57568359375, -0.82666015625, 2.12890625, -1.15234375, 0.97021484375, 1.1064453125, -0.0997314453125, -0.61865234375, 1.3037109375, 0.156494140625, 1.826171875, -0.9501953125, 0.365234375, 1.7158203125, -0.38427734375, 0.0113677978515625, 0.1671142578125, -0.7587890625, 0.129150390625, -0.75390625, 0.5302734375, 1.263671875, 0.82421875, -0.376953125, 0.058929443359375, 0.306640625, 1.078125, -1.703125, 1.515625, 0.88671875, -1.693359375, 1.4404296875, -0.06585693359375, -1.5302734375, 2.390625, -1.537109375, 0.092041015625, -0.393310546875, -0.31884765625, 0.1593017578125, 0.423583984375, 0.426513671875, -0.4013671875, -0.203369140625, 0.1480712890625, -0.88720703125, 0.437255859375, 1.130859375, 1.1884765625, -1.5634765625, 0.384765625, 0.3046875, -1.2998046875, 2.33984375, -1.2529296875, 0.01751708984375, -1.724609375, -1.25, 0.09295654296875, -0.69189453125, -0.63427734375, 1.583984375, 0.15380859375, -1.6572265625, -0.80908203125, 1.19921875, 0.55419921875, 1.0478515625, 1.9287109375, -1.623046875, -0.75146484375, 0.1563720703125, -0.359619140625, -1.1533203125, 0.055908203125, 1.55859375, -0.431396484375, -0.478515625, -0.9599609375, 0.46240234375, -0.77880859375, -1.1484375, 2.369140625, -0.68701171875, 1.453125, -0.1417236328125, 1.5263671875, -0.138916015625, -0.002010345458984375, -0.66162109375, 0.31640625, 0.3466796875, -1.080078125, 0.316162109375, -1.5576171875, 0.352783203125, 0.59326171875, 1.3212890625, -1.0888671875, -0.06768798828125, -2.396484375, 0.92919921875, -0.1983642578125, 0.456787109375, 0.04156494140625, 1.0625, 0.425537109375, 0.13623046875, 0.802734375, 0.79931640625, -1.53515625, -0.272216796875, -0.12939453125, -1.4345703125, 1.140625, -1.2509765625, -1.177734375, 0.492919921875, 1.4970703125, -1.0546875, 1.302734375, -0.053863525390625, 0.4990234375, 0.79638671875, 1.4501953125, -0.396240234375, -2.5703125, -0.60595703125, -0.62158203125, -0.537109375, 0.40869140625, 0.051422119140625, -0.2666015625, -0.312744140625, -0.2498779296875, 0.313720703125, 0.159423828125, -1.361328125, 1.484375, 0.12005615234375, 0.414306640625, -0.0304718017578125, 0.1666259765625, -0.228271484375, -1.67578125, 0.08526611328125, -1.337890625, -0.04901123046875, -0.083984375, -0.262451171875, -0.2197265625, -0.62744140625, 1.87890625, 1.9814453125, -0.0251312255859375, -0.09490966796875, 1.3291015625, -0.0185699462890625, 1.49609375, 0.79443359375, -1.4638671875, -0.25732421875, -0.76806640625, 1.333984375, 0.0760498046875, 0.184326171875, -0.615234375, 1.71484375, 0.1248779296875, -0.168701171875, 1.064453125, -1.427734375, -0.841796875, 1.181640625, 0.51171875, -1.0576171875, 0.91552734375, 0.62255859375, 0.426513671875, -1.6494140625, -0.407470703125, 1.134765625, 0.69482421875, -0.306640625, -0.17724609375, 0.2235107421875, 0.049224853515625, 0.93994140625, 0.79833984375, -0.2108154296875, 1.12109375, 0.53857421875, 1.5458984375, -0.5546875, -0.1824951171875, 1.052734375, -0.56103515625, -0.37158203125, -0.386474609375, -0.3974609375, -1.0322265625, -0.60498046875, 0.884765625, 0.2476806640625, -1.7001953125, -0.00730133056640625, 0.1927490234375, 0.38037109375, -0.465576171875, 0.39697265625, 0.482421875, -0.69140625, 0.01419830322265625, -1.2841796875, 0.43603515625, -0.334716796875, -1.7509765625, -0.04962158203125, 0.73828125, 0.97802734375, 0.07135009765625, -0.072021484375, 0.1751708984375, 0.86669921875, 1.1435546875, 0.043365478515625, -0.9013671875, 1.32421875, -0.1292724609375, 2.041015625, -1.267578125, 0.064208984375, -1.5224609375, -1.65625, 0.47412109375, -0.08978271484375, 0.779296875, 0.93310546875, 0.385009765625, 0.66455078125, 0.84326171875, 0.68896484375, -1.2666015625, -0.61767578125, 1.09765625, 0.50634765625, 1.2216796875, 1.4970703125, -0.277099609375, -0.99169921875, 0.492431640625, 0.111083984375, -0.25439453125, 1.8134765625, 0.396484375, -0.9140625, 0.415283203125, -1.0400390625, -0.050811767578125, -1.12890625, 0.158935546875, 0.0887451171875, -0.314697265625, 0.07177734375, -1.0185546875, 2.08984375, 0.86962890625, 0.88330078125, -0.2880859375, -0.5634765625, 0.61376953125, -1.7568359375, 1.6552734375, -0.04547119140625, -2.37890625, -0.37158203125, 0.681640625, -2.599609375, -0.352783203125, 1.39453125, -0.73681640625, -1.45703125, -0.58740234375, 2.490234375, 1.3544921875, 1.0654296875, -0.771484375, -0.31982421875, 0.2445068359375, 1.1875, -0.497314453125, -0.76025390625, 0.880859375, 1.611328125, 0.4794921875, -1.0546875, -0.429931640625, -0.03631591796875, 0.56640625, -0.24169921875, -1.669921875, -0.326416015625, -0.09332275390625, -1.1953125, -0.138671875, -0.56494140625, -0.6845703125, 2.435546875, -1.1259765625, -0.341064453125, 0.75244140625, 0.9697265625, 0.51513671875, -2.4453125, -0.21728515625, 0.9501953125, 1.1396484375, -0.388427734375, 1.23046875, 1.0654296875, 0.82958984375, 0.45751953125, 0.916015625, 0.3798828125, 0.53369140625, -0.18896484375, 0.0179901123046875, 0.814453125, -0.19970703125, 0.3193359375, -0.57763671875, -0.10076904296875, -0.479736328125, -0.07708740234375, -1.3076171875, 0.83154296875, 0.5966796875, 0.491455078125, -0.87109375, -0.1475830078125, -0.1788330078125, -1.4384765625, -0.26708984375, -1.1015625, 0.1612548828125, -0.591796875, 1.24609375, 1.2451171875, 1.318359375, 2.15234375, -0.70556640625, -0.59765625, 1.5263671875, 0.1380615234375, -1.5029296875, -0.166748046875, -0.066162109375, -0.1923828125, -0.31201171875, -0.450439453125, -0.431640625, 0.52490234375, -0.43701171875, 1.138671875, -0.0946044921875, -1.1337890625, -1.4619140625, 2.58984375, 1.095703125, -0.58154296875, -0.037841796875, 0.08935546875, -0.568359375, 1.0302734375, -0.2315673828125, 0.034393310546875, -1.9990234375, -0.035369873046875, -0.55859375, 0.296142578125, -0.6396484375, -1.7890625, -0.6162109375, -0.380126953125, 1.27734375, 1.361328125, 0.295654296875, 0.03790283203125, -1.888671875, 1.4677734375, -0.9453125, -1.302734375, -1.1396484375, -1.4677734375, -1.453125, -0.47802734375, 0.1785888671875, -1.6435546875, -0.408935546875, -0.338623046875, 0.64892578125, -0.57958984375, -0.395263671875, -0.320068359375, 0.52587890625, 1.98046875, 0.1273193359375, -0.8994140625, -0.68505859375, 0.56396484375, -0.67236328125, 0.48876953125, -1.6845703125, -1.439453125, -0.67333984375, -1.462890625, -1.12109375, 0.1600341796875, 0.0135650634765625, 1.6552734375, 0.3388671875, 1.634765625, -0.40869140625, 0.032745361328125, -0.70556640625, 1.39453125, 1.3134765625, 1.8798828125, -0.09356689453125, -0.54296875, 1.3486328125, -0.6875, 0.019744873046875, 0.289794921875, -0.11480712890625, -0.28125, 0.31884765625, 0.05609130859375, -0.301513671875, 1.1689453125, -0.2169189453125, 0.11492919921875, -0.72607421875, -0.026611328125, 0.017608642578125, 0.43505859375, 2.55859375, -0.95263671875, 0.0867919921875, 0.216796875, -1.2109375, 2.083984375, -1.1376953125, 0.442626953125, 0.57177734375, 0.7919921875, -0.884765625, 0.292236328125, 0.350341796875, 0.282958984375, -0.81982421875, -0.30810546875, -0.375244140625, -0.6484375, -0.8125, 0.499267578125, 0.409423828125, -0.151611328125, 0.003261566162109375, -1.8076171875, -0.139404296875, -0.17041015625, 1.7099609375, -0.447509765625, -0.1485595703125, 0.52197265625, 0.09326171875, -0.9306640625, -0.070068359375, -0.35009765625, -0.77392578125, -0.319580078125, -0.01213836669921875, 0.79150390625, -1.044921875, -0.428466796875, -2.142578125, 2.08203125, 0.97509765625, -0.73828125, 0.20751953125, 0.86474609375, -0.285400390625, 0.198974609375, 1.0859375, -0.25048828125, -0.85009765625, 0.361328125, -0.05633544921875, -1.349609375, -1.2431640625, -0.03533935546875, -0.7841796875, -1.1513671875, -0.334716796875, -0.456298828125, 0.326171875, -2.232421875, -1.0859375, 1.3349609375, -0.8359375, 0.1533203125, 0.64404296875, 0.98779296875, -0.0830078125, -1.49609375, 0.129150390625, 0.364501953125, 0.07794189453125, -1.2763671875, 0.52197265625, -1.41015625, 0.3154296875, -0.2489013671875, -0.974609375, -0.041961669921875, 0.77734375, -0.2371826171875, -0.888671875, -0.1270751953125, 0.57763671875, 0.615234375, 0.51708984375, -0.093017578125, -1.544921875, -0.405029296875, -0.796875, 0.1890869140625, -1.6640625, 0.2049560546875, 3.1484375, 1.013671875, 0.08673095703125, -0.6875, -2.17578125, 0.4912109375, 1.9970703125, 0.305419921875, -0.57861328125, -0.301025390625, 0.0750732421875, -0.91162109375, -0.381591796875, -1.240234375, -0.54248046875, -0.0531005859375, -2.38671875, 1.4169921875, -1.673828125, -0.1102294921875, -0.87158203125, 0.2218017578125, 1.365234375, -0.323486328125, 1.4619140625, 0.253662109375, 0.96484375, -0.423828125, 0.56591796875, -0.88232421875, -2.509765625, 1.4970703125, 0.8232421875, -0.9658203125, 0.1611328125, -0.69189453125, 1.625, 0.08074951171875, -1.27734375, 0.40234375, 0.55615234375, 0.0858154296875, 0.2008056640625, 0.367919921875, 0.34228515625, 0.69287109375, -0.53955078125, 0.112548828125, 0.12310791015625, 0.1507568359375, 0.6484375, 1.056640625, 0.006053924560546875, -0.74365234375, -1.4619140625, 0.11236572265625, 0.281982421875, -0.323974609375, -0.224609375, -0.609375, -0.67041015625, -0.81494140625, 0.7548828125, 1.16796875, -0.8369140625, 0.732421875, 1.9912109375, -0.78125, -1.0634765625, -2.466796875, -0.66796875, -0.736328125, 1.07421875, -0.78759765625, 0.97509765625, -2.15234375, 0.59619140625, 0.09716796875, -1.3037109375, -0.280029296875, -0.321044921875, -1.03515625, 1.1982421875, 0.0438232421875, -0.44873046875, 0.367431640625, -0.498291015625, -0.44873046875, -0.640625, -0.441162109375, 0.29150390625, -0.95556640625, 0.374755859375, 0.916015625, -0.032623291015625, -1.892578125, -0.061767578125, 2.037109375, -0.88037109375, 0.0635986328125, 0.317626953125, -0.77978515625, 1.0546875, -0.83349609375, -1.7255859375, -1.08203125, 2.11328125, 1.326171875, -0.66943359375, -1.5830078125, -0.966796875, -0.77880859375, -1.0107421875, -1.3720703125, -0.0504150390625, -0.572265625, 0.87451171875, -0.12432861328125, 1.2939453125, -0.529296875, -0.93701171875, 1.1025390625, 0.37646484375, -0.35498046875, -0.60107421875, -1.23046875, -0.50244140625, 0.0244140625, 0.572265625, -0.78955078125, 0.58203125, 0.681640625, -1.009765625, 0.333984375, -1.2841796875, -0.71240234375, 0.2685546875, 0.320068359375, -0.390625, -0.97412109375, -1.2216796875, 0.0267333984375, -1.1513671875, -2.7734375, 2.126953125, 0.841796875, -0.58203125, 0.9404296875, -1.5244140625, -0.2171630859375, 0.61572265625, -0.86572265625, -0.94482421875, 0.8740234375, 0.642578125, -1.466796875, 0.080078125, -1.1953125, 0.7099609375, -0.138916015625, 1.0693359375, -0.28955078125, -0.67529296875, -0.159912109375, -1.4423828125, -0.0775146484375, 0.3515625, 0.393798828125, 0.61181640625, -1.60546875, 1.3369140625, 1.197265625, 0.443359375, -0.301513671875, -0.6767578125, 0.1500244140625, -1.4912109375, -0.397705078125, -2.966796875, 1.9296875, 0.8740234375, -0.66845703125, 0.66845703125, -0.11260986328125, -0.505859375, -0.8984375, 0.73583984375, -0.765625, -0.93798828125, -0.62744140625, 0.93603515625, 1.0986328125, -1.138671875, -1.658203125, 1.6259765625, -0.1983642578125, -0.445556640625, -1.3193359375, 1.93359375, -0.96923828125, 1.1787109375, -0.0787353515625, 0.83544921875, -0.10882568359375, 0.90673828125, 1.177734375, 2.87109375, 0.76123046875, 0.2060546875, -0.09149169921875, -0.89990234375, -1.4462890625, 0.15625, 0.146728515625, 0.73095703125, 0.023040771484375, -0.609375, -1.107421875, -0.77587890625, -0.85791015625, 0.4677734375, 0.50830078125, -0.03570556640625, -0.253173828125, 0.483154296875, 0.14306640625, 1.87890625, 0.413818359375, 0.70703125, -0.385498046875, -2.2578125, 0.445556640625, 0.341064453125, 0.2313232421875, -2.470703125, 2.005859375, 2.181640625, 0.6953125, 1.6240234375, 0.5595703125, 0.5791015625, 0.96337890625, -0.2252197265625, -0.0634765625, -0.280517578125, -2.19140625, 1.0849609375, 0.65771484375, 1.404296875, 1.31640625, -0.60986328125, -1.6123046875, -1.3134765625, -0.189697265625, -0.385986328125, -0.79345703125, -0.09033203125, -1.5986328125, -0.9775390625, 0.038604736328125, 1.14453125, -1.06640625, -1.5390625, -1.6728515625, -1.67578125, 0.0219573974609375, -1.205078125, -0.93212890625, -0.253662109375, 1.2626953125, -0.23291015625, 1.08203125, -0.673828125, -0.09771728515625, -0.9580078125, -0.35986328125, 1.7255859375, -0.412841796875, 1.2216796875, 0.467041015625, -2.13671875, -0.296142578125, -1.658203125, -1.5595703125, -0.468505859375, 0.1324462890625, -0.85888671875, 1.623046875, -0.332763671875, 0.5107421875, -1.4208984375, -0.70263671875, -0.18115234375, 0.53857421875, -0.019134521484375, -1.3388671875, 1.4765625, 1.0380859375, 1.3115234375, 2.12890625, 0.53857421875, 1.322265625, 0.37890625, 0.11590576171875, 0.83349609375, 2.134765625, 0.083251953125, 1.107421875, 0.5830078125, -0.05078125, 0.69677734375, 1.5810546875, -0.86669921875, -0.5283203125, 0.9765625, -1.8232421875, -1.5, 0.221435546875, 0.026824951171875, 1.0458984375, 0.37939453125, -0.1278076171875, 0.845703125, -0.42529296875, -1.056640625, -0.452392578125, 1.5556640625, -1.673828125, -1.8701171875, 1.044921875, 0.72412109375, 1.3046875, 0.255126953125, -0.56591796875, -0.7841796875, 0.98486328125, 0.353759765625, -0.16015625, -1.357421875, -1.0, 1.2568359375, 0.7685546875, 1.51171875, -1.01171875, 0.6435546875, -0.396240234375, -0.131591796875, -0.94921875, -0.2666015625, -0.66357421875, 1.46484375, 0.44775390625, 0.77294921875, 0.28662109375, -1.6259765625, 0.45751953125, 1.39453125, -0.0220489501953125, -0.094482421875, -0.5537109375, 1.46484375, -1.1611328125, 0.392578125, 2.16015625, -2.392578125, 0.16650390625, -0.29296875, -1.798828125, 0.9423828125, -0.0275421142578125, 0.4736328125, -0.708984375, 1.615234375, -1.5673828125, 0.5732421875, -1.3291015625, 0.86669921875, -0.62548828125, 0.80810546875, -1.6484375, 1.10546875, -1.4033203125, 1.712890625, -0.943359375, -1.5595703125, 1.0751953125, -1.2421875, -0.58837890625, 1.560546875, -0.56982421875, 1.3095703125, 0.269775390625, 0.06658935546875, -0.53857421875, 0.288818359375, -0.14892578125, -0.344970703125, -0.2841796875, -0.310791015625, 0.2822265625, 0.322998046875, 0.8388671875, -0.1514892578125, -1.6064453125, 0.677734375, 0.78125, -0.260498046875, -0.476318359375, 1.58203125, 1.12109375, -0.400146484375, 1.529296875, -1.1005859375, 0.96435546875, 1.59375, -0.310546875, 1.6611328125, 0.60791015625, 0.4453125, 0.53125, -0.12322998046875, 0.04864501953125, -1.0107421875, 0.259033203125, -0.09515380859375, 1.4375, -0.17333984375, 1.1396484375, -0.59326171875, 0.8671875, -1.2744140625, 2.328125, -1.171875, -1.98046875, -0.151123046875, 1.0869140625, -0.72314453125, 0.44091796875, -1.041015625, -0.1964111328125, -0.0560302734375, -1.04296875, 0.75634765625, 0.2242431640625, 0.6787109375, -1.3447265625, -0.18115234375, -0.84423828125, 0.74658203125, 0.56591796875, -2.400390625, -0.935546875, 0.94580078125, -0.227783203125, 0.38037109375, 1.0576171875, 2.921875, 0.375244140625, -1.1171875, 1.6513671875, 0.123291015625, 0.93017578125, 0.57373046875, 0.9521484375, 1.23046875, 0.728515625, 0.76953125, 0.57470703125, -0.23974609375, -1.1787109375, -0.9140625, -1.732421875, 0.54248046875, -0.75, -0.2017822265625, -1.185546875, -1.1416015625, 0.7197265625, 0.12457275390625, -0.826171875, -0.1514892578125, 1.2158203125, -1.23828125, -0.12432861328125, -0.87451171875, -0.92724609375, 0.708984375, -1.328125, -0.5712890625, 0.433349609375, 0.301025390625, -1.513671875, 1.1865234375, -0.1512451171875, -0.7998046875, 1.7021484375, -1.923828125, -0.208984375, -0.60498046875, -0.468994140625, -0.71728515625, 0.794921875, 0.15087890625, -0.002384185791015625, 0.07427978515625, -0.51416015625, 0.74755859375, 0.487548828125, -1.529296875, -0.525390625, 0.67529296875, -0.72314453125, 0.114013671875, 0.189453125, 0.19189453125, 0.1348876953125, 0.36865234375, 0.53515625, 0.90087890625, -0.66796875, -1.4990234375, 0.1917724609375, 1.1748046875, -1.8046875, -0.0015916824340820312, 1.3017578125, -0.448974609375, 1.7451171875, -0.2332763671875, 0.400390625, 1.1640625, 0.208740234375, 0.328369140625, 0.513671875, 1.390625, 0.80078125, 0.76025390625, -0.89453125, 0.29296875, -0.293701171875, 0.6962890625, -1.46875, -0.0792236328125, 0.662109375, -0.810546875, -0.8203125, -1.1455078125, -2.1640625, 0.744140625, 1.046875, 1.1962890625, 0.2724609375, -1.43359375, -0.234619140625, 0.537109375, -1.2109375, -0.0286407470703125, -1.591796875, 0.0606689453125, -0.63134765625, -1.5078125, -1.5556640625, 1.06640625, -1.4560546875, 1.85546875, 0.274169921875, 0.2958984375, 0.11334228515625, 0.53564453125, -2.5078125, 0.3369140625, -0.7197265625, 1.2021484375, 2.08984375, 0.028411865234375, 1.611328125, -0.025238037109375, 0.209228515625, -0.04681396484375, 0.3076171875, 3.087890625, -1.419921875, -0.51416015625, 0.923828125, 1.603515625, -0.9716796875, -0.60595703125, 0.2327880859375, -0.78466796875, -0.216552734375, 0.265380859375, 0.00750732421875, 0.49560546875, 0.260498046875, 0.2958984375, 1.544921875, 0.233642578125, 0.299072265625, -1.6630859375, -0.33056640625, 0.978515625, -0.56298828125, -0.6591796875, -0.459716796875, 0.259521484375, -1.212890625, -0.11627197265625, 1.3466796875, -0.4453125, 0.476318359375, -0.261962890625, 1.92578125, 0.042388916015625, 0.73095703125, 1.689453125, -1.2080078125, 0.52294921875, 0.377197265625, -0.6650390625, -1.1474609375, 0.10760498046875, -0.07659912109375, 0.490966796875, -1.75, 0.7802734375, 0.01160430908203125, -0.89990234375, 0.144287109375, 0.15283203125, -1.8330078125, -0.61962890625, 0.386474609375, 0.625, -1.4306640625, 0.8154296875, 0.466552734375, -0.28515625, -1.734375, -2.337890625, -1.1396484375, 0.90380859375, -0.15380859375, -0.982421875, -0.57373046875, 2.236328125, -1.1474609375, -1.7705078125, -1.068359375, 2.296875, -0.361083984375, 0.265869140625, 0.6865234375, 1.6396484375, -0.55126953125, -0.9443359375, -0.058837890625, 0.356689453125, 0.904296875, -0.0194854736328125, -0.74072265625, -2.9296875, 2.859375, 1.1162109375, -0.9130859375, -0.4658203125, 0.167724609375, 0.457763671875, 0.69677734375, -1.3720703125, 0.1353759765625, 1.43359375, -0.7197265625, -1.16796875, 0.469970703125, 0.50390625, 0.68310546875, 0.04791259765625, -0.75927734375, -1.19921875, 0.80029296875, 1.96875, -1.419921875, 1.1015625, 1.58984375, -2.2890625, 0.0081024169921875, 0.445556640625, 1.294921875, 1.0537109375, 0.078857421875, 0.05615234375, -2.03125, 0.2279052734375, 0.250732421875, -1.9873046875, -0.90625, 1.2353515625, -0.005237579345703125, -0.130615234375, -1.56640625, -0.775390625, 0.1754150390625, 0.80224609375, 0.383056640625, -0.0438232421875, 0.388671875, -0.9404296875, 0.76513671875, 1.1259765625, -0.7021484375, 0.90283203125, 0.65380859375, 0.21630859375, 0.173828125, -0.6767578125, 1.3291015625, 0.13134765625, -0.6669921875, -0.3876953125, 0.10345458984375, -0.3994140625, 0.6826171875, 0.5927734375, -1.421875, -0.1480712890625, -1.005859375, -1.400390625, 1.5927734375, 0.716796875, -2.13671875, 1.4462890625, 1.302734375, -0.00948333740234375, 0.76123046875, -0.6162109375, -0.0127105712890625, -1.806640625, 1.7431640625, -1.193359375, -0.85693359375, -0.65576171875, -0.03460693359375, -0.76806640625, -0.48828125, -0.376708984375, -0.0865478515625, 2.736328125, -1.4765625, 1.2685546875, 0.83935546875, 0.484619140625, 0.1644287109375, 0.2998046875, -0.35107421875, -2.873046875, 0.9375, 1.357421875, 0.004638671875, -1.580078125, 1.33203125, 0.380859375, -0.57421875, 0.0204010009765625, 0.91455078125, 0.0068817138671875, 0.9453125, 0.61572265625, 0.76318359375, -0.025299072265625, 0.150390625, -0.35205078125, 1.96875, 1.1162109375, 0.60546875, -0.2081298828125, -0.26708984375, -0.471923828125, -0.96728515625, -0.20263671875, 0.44189453125, 0.342529296875, 1.1689453125, -1.3974609375, -1.4208984375, 0.83935546875, 0.8525390625, -2.0078125, 1.55078125, 1.056640625, 0.80419921875, 1.25390625, 1.6142578125, -1.466796875, 0.77783203125, -0.767578125, 1.04296875, 0.444091796875, -0.499755859375, -0.9814453125, -0.1568603515625, 0.84423828125, 0.609375, -1.4033203125, -1.388671875, 0.312744140625, 0.51806640625, -0.73974609375, -0.69091796875, -1.2900390625, -1.5625, 0.9736328125, 0.677734375, -0.09588623046875, 0.257080078125, 0.67724609375, 2.1171875, -1.69921875, 0.7138671875, -0.337646484375, 2.29296875, 0.2216796875, 0.935546875, -0.296630859375, 0.10443115234375, -0.935546875, -0.59521484375, -0.4208984375, -0.6787109375, 1.927734375, 0.7783203125, 0.1199951171875, 2.17578125, 0.440673828125, 0.62255859375, 0.498291015625, 0.2783203125, -1.6142578125, 0.39111328125, 1.609375, 0.58447265625, -1.36328125, 0.37841796875, 0.64013671875, -0.30322265625, 1.115234375, 1.9462890625, 0.1627197265625, -0.2125244140625, 1.0439453125, 0.921875, 0.48681640625, 3.62890625, 1.3447265625, 0.06976318359375, -0.4501953125, -0.78369140625, -0.042083740234375, -0.10693359375, 0.8310546875, -0.246826171875, -0.01192474365234375, 0.62744140625, -0.8134765625, -0.9814453125, -0.9697265625, 0.1663818359375, 0.2626953125, 0.032257080078125, -0.96484375, -0.92626953125, -1.19140625, 2.43359375, 1.7666015625, 0.3125, -0.615234375, -2.07421875, -0.9326171875, -0.257080078125, -2.77734375, -0.49853515625, 0.293701171875, 1.2646484375, -0.418701171875, -0.387939453125, -0.87109375, -0.1317138671875, -0.35693359375, -0.95751953125, -1.4609375, -2.732421875, 0.0035190582275390625, -1.2109375, 0.149658203125, -0.1627197265625, 0.57568359375, 1.390625, 0.0765380859375, 1.2939453125, -0.328857421875, 0.05963134765625, 1.42578125, 0.255859375, -1.05859375, 1.017578125, -0.826171875, -0.07666015625, -1.6083984375, 0.266357421875, -0.04583740234375, 1.2451171875, 1.232421875, -1.068359375, -0.226318359375, 0.9970703125, -0.251708984375, 0.6328125, -1.4375, -0.36669921875, 0.1478271484375, -2.2578125, -1.8857421875, 0.1446533203125, 0.94384765625, -0.73974609375, 0.33447265625, 0.71484375, -0.873046875, -1.666015625, 1.6103515625, -0.1405029296875, 0.86279296875, -0.415283203125, -0.0032215118408203125, 0.44384765625, -0.70166015625, -0.41015625, 3.29296875, 0.693359375, -0.10040283203125, 1.3291015625, 1.2451171875, 2.193359375, -0.32080078125, -1.15625, 0.0595703125, 0.8369140625, 2.916015625, 1.3291015625, 2.01171875, 0.685546875, 0.03582763671875, -0.52294921875, -0.57958984375, 1.6845703125, -0.6962890625, 0.9365234375, -2.92578125, 0.3076171875, 0.151611328125, -2.263671875, 0.007366180419921875, 0.76904296875, -1.169921875, 1.2734375, 0.46484375, 1.7373046875, -1.4541015625, -0.07977294921875, 0.28662109375, -0.833984375, -0.52685546875, -0.38427734375, 0.037933349609375, 0.415771484375, -0.054656982421875, 0.1044921875, -0.1783447265625, -1.4599609375, 0.7490234375, 0.64306640625, 0.471923828125, 1.765625, 1.51953125, -0.52294921875, 1.3076171875, -1.12109375, -0.297607421875, 0.314208984375, -0.41552734375, 0.52294921875, 0.40673828125, 2.037109375, 1.021484375, -0.2188720703125, 0.432861328125, 0.1534423828125, 0.037872314453125, -1.5341796875, -0.412353515625, -0.6796875, -0.32861328125, -2.546875, -0.6708984375, 0.67626953125, 0.60986328125, 0.2493896484375, -0.1978759765625, 0.4375, 0.70947265625, 1.25, -1.201171875, 0.505859375, -2.693359375, -1.4697265625, 0.91796875, 0.393310546875, -0.1888427734375, -1.25, -0.0096588134765625, -1.4794921875, 1.5029296875, -0.7041015625, -0.460693359375, -0.140869140625, -0.62158203125, -0.88037109375, -0.454833984375, -1.5732421875, 0.54296875, 1.4599609375, 0.2415771484375, 1.2275390625, 0.92138671875, 0.953125, -1.7783203125, -0.59521484375, 0.473388671875, 2.46875, 0.892578125, 1.1728515625, -1.3837890625, -0.311767578125, -0.362548828125, -0.352294921875, 1.2470703125, 0.78125, -0.6884765625, 0.1505126953125, -1.6953125, -0.47119140625, -0.296875, -1.8369140625, -0.12237548828125, -1.6064453125, -0.50146484375, 0.19091796875, 0.132080078125, -1.2138671875, 0.143310546875, -1.0673828125, -0.61572265625, -0.91357421875, 0.3583984375, -0.0006933212280273438, 0.0731201171875, 0.63720703125, -1.267578125, 0.3837890625, -0.009613037109375, -0.81005859375, -1.2939453125, -1.3720703125, -0.75537109375, -1.9521484375, 1.3671875, 2.318359375, 0.62744140625, -1.1650390625, -0.5947265625, 1.9208984375, -0.0042877197265625, 0.5751953125, 0.8603515625, 0.96044921875, -0.51123046875, 1.634765625, -0.45849609375, -0.59130859375, 0.2005615234375, 0.1221923828125, -0.3447265625, 0.2498779296875, 0.475341796875, -0.0850830078125, 0.5537109375, 0.37939453125, 0.66845703125, 0.9345703125, 0.681640625, 1.546875, 0.93994140625, 1.1669921875, 0.311279296875, 0.91650390625, -0.564453125, -2.640625, 0.71630859375, -0.83984375, 0.1571044921875, 0.4140625, 0.42578125, -1.2099609375, -0.58349609375, 0.1864013671875, 0.367431640625, 0.17626953125, 0.351318359375, 0.55908203125, 1.1455078125, -1.1630859375, 0.8740234375, 0.5205078125, -0.359130859375, -1.4248046875, 0.1243896484375, -0.022308349609375, 0.0034732818603515625, -0.1107177734375, 0.59521484375, 1.982421875, 1.0322265625, 0.96044921875, -1.3408203125, 1.5712890625, 1.1298828125, -1.2158203125, -0.52392578125, 0.332763671875, -0.418701171875, -0.27490234375, 1.2900390625, 0.71630859375, -0.779296875, 0.035797119140625, -1.6826171875, 0.58447265625, -0.6787109375, 1.4736328125, 0.2215576171875, -1.224609375, 0.1947021484375, -1.64453125, 0.154296875, 0.52294921875, 0.29638671875, -1.4423828125, 0.239501953125, 0.50146484375, -1.2353515625, -0.759765625, 2.548828125, 1.0537109375, 0.5634765625, 2.033203125, 0.7705078125, 1.1796875, 0.7451171875, 0.1583251953125, -0.48583984375, -0.18017578125, -0.12347412109375, 0.062744140625, -1.0693359375, -2.427734375, -0.427490234375, 0.66845703125, 1.576171875, -1.9287109375, -0.7001953125, 1.416015625, 0.1981201171875, 0.427001953125, 0.93994140625, -1.822265625, -0.92431640625, 0.82421875, 1.51171875, 0.61279296875, 0.78466796875, 1.44140625, 0.1536865234375, -0.40185546875, 0.062225341796875, -0.4990234375, -0.2330322265625, 0.252685546875, -0.483642578125, -1.8974609375, -0.46142578125, 0.7158203125, 1.2353515625, -0.1571044921875, 0.599609375, 0.157470703125, -1.44140625, -2.044921875, -0.90087890625, 0.10321044921875, -0.73388671875, -0.3193359375, 0.25634765625, -1.2958984375, -1.1572265625, 0.68603515625, 2.130859375, 0.486572265625, -0.0209197998046875, 0.447265625, -0.95556640625, -0.07586669921875, 0.57421875, 1.5693359375, -0.953125, 0.65771484375, -0.410400390625, -1.1708984375, 1.677734375, -0.060272216796875, 0.5927734375, 1.0654296875, 0.74267578125, 0.73046875, 0.09942626953125, 1.8447265625, 0.5009765625, -0.52587890625, -0.0948486328125, -0.296630859375, -0.0304718017578125, 1.4619140625, 0.65966796875, -1.4521484375, -1.5712890625, -1.4453125, -1.1689453125, -1.10546875, 0.66650390625, 0.058685302734375, -1.3427734375, -0.68017578125, 0.039306640625, -0.182861328125, 1.294921875, 1.1279296875, -1.5009765625, 0.420654296875, -0.8701171875, -0.64990234375, 0.26708984375, -0.050628662109375, 0.78076171875, -2.15234375, -0.486572265625, 0.03607177734375, -0.2734375, 1.8994140625, 0.45947265625, 0.50830078125, -0.218017578125, -1.45703125, 0.078125, 2.390625, 0.123046875, -0.3935546875, -2.484375, 0.755859375, 1.4228515625, 0.7431640625, -0.99853515625, 0.60302734375, 0.14794921875, 0.2705078125, 0.232177734375, -0.57275390625, -1.8828125, -1.466796875, -1.021484375, -2.474609375, -0.188232421875, -0.039825439453125, -1.6533203125, 0.724609375, 1.201171875, -0.322021484375, -0.2059326171875, -1.0244140625, -0.478759765625, 1.224609375, 0.01580810546875, -0.487060546875, -0.59375, 0.61669921875, 0.56591796875, 0.318603515625, 0.537109375, -0.841796875, 2.140625, -1.791015625, 0.494873046875, -0.830078125, -0.281494140625, 0.432373046875, -0.1259765625, 1.6767578125, -0.275146484375, -1.00390625, 1.8251953125, -0.1968994140625, -0.1439208984375, 1.501953125, -1.1484375, 2.638671875, -1.1083984375, 0.8681640625, 0.1182861328125, -0.234619140625, 0.453857421875, -0.4677734375, 0.352783203125, 0.53564453125, 0.1688232421875, 0.1702880859375, 1.908203125, -0.14794921875, 3.0390625, -0.3408203125, -0.1561279296875, -0.4619140625, 0.06494140625, -0.79248046875, -0.89013671875, 0.2445068359375, 0.179443359375, -0.08734130859375, 0.42578125, -0.430419921875, -1.46484375, -1.275390625, 0.4228515625, -0.15625, -0.91064453125, 0.57763671875, -1.2880859375, 0.14208984375, 0.7734375, -0.703125, -0.446044921875, 0.182861328125, 2.919921875, 0.53271484375, -0.006679534912109375, -0.73779296875, 0.415283203125, 0.2255859375, 0.10333251953125, -0.0176239013671875, -0.72998046875, -0.291015625, -0.521484375, -0.80712890625, -0.80029296875, 0.40283203125, 0.1043701171875, -0.279541015625, -0.50537109375, 1.189453125, 0.07598876953125, 0.5205078125, 0.1934814453125, 0.11419677734375, 0.64013671875, 0.34130859375, -1.216796875, -0.1820068359375, -0.63037109375, 1.0087890625, -0.55615234375, 0.50537109375, 1.033203125, 0.267578125, 0.869140625, 1.0068359375, -0.71044921875, -0.85791015625, -0.5546875, 0.412353515625, -0.380859375, -0.10382080078125, 0.4853515625, 1.4296875, 1.349609375, -0.33984375, -1.13671875, 1.1318359375, -0.357177734375, -1.0068359375, -0.11151123046875, -0.12139892578125, -0.60498046875, -0.7919921875, 1.0107421875, -0.279541015625, 0.091064453125, 0.364501953125, 0.859375, 0.34814453125, -0.356201171875, 0.75, 0.57177734375, 0.98486328125, -0.77978515625, -1.2822265625, -0.1715087890625, -0.56005859375, -1.943359375, 1.1845703125, -2.275390625, -0.4384765625, 0.1455078125, 0.88623046875, 0.908203125, 1.5439453125, 0.66015625, 0.23486328125, -0.904296875, -0.05413818359375, 0.81689453125, -0.1314697265625, -1.8017578125, -0.450927734375, -0.962890625, 0.346435546875, 0.4677734375, -1.4873046875, 0.5517578125, 0.03216552734375, 1.119140625, 0.57861328125, -0.06011962890625, 0.294921875, -0.8857421875, -0.45361328125, -0.55615234375, -0.27099609375, 0.87353515625, 0.642578125, -0.8388671875, -0.442138671875, 1.669921875, -0.9150390625, 1.904296875, -1.5, 0.1636962890625, 0.300048828125, -0.826171875, -0.58837890625, -1.32421875, 0.7138671875, -1.3173828125, 0.2313232421875, 1.2646484375, 1.2451171875, -0.215087890625, 0.26025390625, -0.0255889892578125, 1.7802734375, -0.53271484375, 2.12890625, -0.0421142578125, 0.44091796875, -0.2841796875, -0.311767578125, -0.1334228515625, -1.5703125, -0.62109375, 0.62060546875, -0.505859375, 1.0107421875, -2.50390625, -0.921875, -0.7138671875, 0.320068359375, 1.712890625, 1.595703125, 1.474609375, -0.15234375, 1.474609375, 0.59130859375, 1.48046875, 0.08001708984375, 0.442626953125, 0.57861328125, 2.060546875, 0.2335205078125, 0.71923828125, 0.281494140625, 1.4443359375, -0.086181640625, -1.1396484375, 1.3837890625, 0.77197265625, -0.334716796875, 0.80908203125, 1.0126953125, 0.9658203125, 0.283203125, 0.4609375, -0.73779296875, -0.6220703125, 3.333984375, -0.74462890625, -0.810546875, -1.7763671875, 1.03515625, -0.85498046875, 0.47607421875, -2.052734375, -1.8994140625, 0.5849609375, 0.6875, -0.267822265625, -0.82666015625, -0.252685546875, -0.8427734375, 0.6005859375, -1.076171875, -1.6064453125, -1.8701171875, -1.271484375, 0.413818359375, -0.748046875, -0.335693359375, 0.57470703125, -0.81298828125, -1.14453125, -0.4697265625, -0.293701171875, -1.3974609375, 0.337158203125, -0.68017578125, 0.93701171875, -1.5302734375, -0.269775390625, -0.6962890625, -0.23095703125, 1.482421875, -1.140625, -0.92138671875, 1.2763671875, -0.355712890625, -0.07861328125, 2.189453125, -1.267578125, 1.2607421875, -1.0166015625, -0.712890625, 0.0102691650390625, 1.236328125, 0.028289794921875, -1.1025390625, -1.0693359375, 0.045379638671875, -0.05914306640625, -1.3271484375, 0.67041015625, 1.3037109375, -1.5439453125, -2.5, 0.049774169921875, 0.0806884765625, -1.0908203125, -0.86767578125, -1.3671875, -0.69970703125, -1.8212890625, -0.311279296875, -1.3876953125, -1.5087890625, -0.174072265625, 0.07470703125, 0.1949462890625, -1.3642578125, 0.5791015625, 0.348876953125, 0.6455078125, -0.72802734375, -0.0794677734375, -1.412109375, -0.1502685546875, -1.0439453125, 0.96435546875, 0.5205078125, 0.44580078125, 0.08941650390625, -0.01154327392578125, -0.15283203125, 0.21484375, 0.2802734375, -0.5546875, -0.57666015625, 1.62890625, -1.951171875, -0.8515625, -0.440673828125, 0.95068359375, -0.134521484375, -0.493408203125, 1.3671875, 0.87890625, 0.63037109375, -1.083984375, 0.677734375, -0.305419921875, -0.1900634765625, -1.1201171875, 1.248046875, 1.119140625, 0.8837890625, -0.448974609375, -0.2017822265625, 1.142578125, 0.2724609375, -0.377685546875, -1.2236328125, 1.3876953125, -0.252197265625, 0.2403564453125, -1.748046875, 1.193359375, 1.3876953125, -1.4287109375, -0.303955078125, 2.025390625, -0.87353515625, 1.05078125, -1.255859375, -1.1337890625, 0.6875, 1.037109375, 0.81494140625, 0.30126953125, 0.79150390625, 0.358642578125, -1.78125, 0.0474853515625, 0.37646484375, -1.0234375, -0.248046875, 0.4326171875, 0.044342041015625, -2.2890625, 0.4794921875, -1.0576171875, 0.12274169921875, 0.09027099609375, 0.583984375, 0.0146331787109375, 1.6484375, -0.767578125, 0.397705078125, 1.044921875, 2.1484375, -0.128662109375, -0.314453125, -0.9599609375, 0.9072265625, 0.350341796875, -0.73095703125, -0.1444091796875, 0.07708740234375, 1.0712890625, 0.05010986328125, 1.6142578125, -1.0234375, -0.04931640625, -0.32958984375, 0.263916015625, -0.44775390625, -1.177734375, 2.087890625, -0.439453125, -1.4560546875, -0.83251953125, 1.064453125, 0.7138671875, -0.70849609375, -1.84375, -2.126953125, -1.0380859375, 0.80908203125, 0.986328125, 2.078125, 0.52685546875, 1.0771484375, -0.19140625, 0.0044403076171875, 0.10809326171875, -0.356201171875, 1.296875, -1.3662109375, -0.2144775390625, -1.0810546875, -1.12890625, -0.7353515625, -0.95654296875, -0.2161865234375, -0.9755859375, 0.310546875, 0.06927490234375, -1.0146484375, -1.2333984375, -0.89697265625, 0.2900390625, -1.1884765625, 2.099609375, 1.263671875, 0.353759765625, 0.9423828125, -0.15673828125, -0.343017578125, -1.2138671875, -1.1796875, -0.058624267578125, 0.97021484375, -0.91259765625, -0.248046875, 0.2099609375, 1.064453125, -0.2178955078125, 0.184326171875, 0.52685546875, 0.83642578125, -1.64453125, 0.2210693359375, 0.525390625, -0.16455078125, -1.3974609375, 0.320556640625, -0.10089111328125, 0.345947265625, -1.6953125, -0.53515625, 0.7919921875, -0.1187744140625, -0.2095947265625, -1.9033203125, -1.4970703125, 0.70703125, -0.91357421875, 1.1533203125, 1.3310546875, -1.994140625, 0.481689453125, 0.513671875, -0.304931640625, 0.173828125, 0.70654296875, 0.61767578125, -1.716796875, 0.446533203125, -1.20703125, -0.95458984375, -0.98046875, 0.591796875, 1.94140625, -1.0263671875, 0.0086212158203125, -1.150390625, 1.5986328125, -0.2423095703125, 0.013763427734375, 1.1650390625, 0.69580078125, 1.3095703125, 1.169921875, 0.65869140625, -2.34375, 1.298828125, 0.9404296875, 0.681640625, 1.9697265625, 0.407470703125, -0.1513671875, 0.1702880859375, 0.82763671875, 1.3037109375, 2.88671875, 1.1845703125, -0.8916015625, -0.66015625, 1.21875, -0.72021484375, -0.0184478759765625, 0.630859375, -0.69384765625, 0.6552734375, 0.4462890625, 0.8935546875, 0.13427734375, -0.060882568359375, -0.69287109375, 0.187744140625, -0.65478515625, -0.0430908203125, 0.52490234375, 1.0732421875, -0.1910400390625, 0.689453125, 0.962890625, 1.6875, 0.079345703125, -1.6181640625, 1.43359375, 0.3427734375, 1.5234375, -0.5478515625, -1.5791015625, -0.6103515625, 0.0745849609375, 0.72900390625, -0.19970703125, -1.568359375, 0.66259765625, -1.0380859375, -1.4130859375, -0.257080078125, -1.6201171875, -1.1650390625, 0.521484375, -0.01363372802734375, 0.0189361572265625, 0.96240234375, -0.83984375, -0.68896484375, 0.671875, -1.2587890625, 0.1583251953125, -1.224609375, 0.7978515625, 0.381103515625, -0.50537109375, -1.4912109375, 0.8974609375, 0.73046875, -0.204345703125, -0.352294921875, -0.158935546875, 0.8046875, 1.2568359375, -0.46240234375, -1.1552734375, 1.3779296875, 1.2587890625, -0.356201171875, 1.943359375, 0.5263671875, -0.234619140625, 0.7099609375, -0.927734375, 1.8212890625, -0.87548828125, 0.603515625, 0.01120758056640625, -0.23583984375, -1.763671875, 0.4091796875, 1.185546875, -1.0, 1.4697265625, 2.01171875, -0.28076171875, 2.490234375, -1.44140625, -2.025390625, -0.453125, 2.1171875, 0.95703125, -0.35498046875, -1.482421875, -0.405517578125, 1.8876953125, -0.414306640625, -0.0751953125, 1.2373046875, -0.99658203125, -2.291015625, 0.424560546875, 0.059478759765625, -1.21875, 0.64697265625, 1.302734375, -0.266357421875, 0.73583984375, -1.072265625, -1.32421875, 0.1475830078125, -0.414306640625, 0.7236328125, -0.33447265625, -0.432861328125, -0.5517578125, -0.463623046875, -0.62109375, -0.361328125, -1.0146484375, -2.3046875, -1.41796875, -0.3232421875, 0.666015625, 0.034271240234375, 0.712890625, -0.7490234375, -0.77392578125, -0.343505859375, 0.463134765625, -0.3544921875, 0.1461181640625, -0.716796875, 0.505859375, 2.244140625, -0.197509765625, -0.5009765625, 1.794921875, -0.955078125, -0.5576171875, -1.08984375, 1.58984375, 0.716796875, -1.333984375, -1.05859375, 0.80859375, -0.281982421875, -0.8505859375, -1.8994140625, 0.46337890625, -0.4716796875, -1.744140625, -0.62841796875, 0.79248046875, -0.73486328125, 0.9677734375, -0.8017578125, 0.1317138671875, -0.73046875, 0.3642578125, 0.280517578125, 0.8505859375, -2.1484375, 2.291015625, -0.37158203125, 1.53515625, -0.0019626617431640625, -0.354736328125, -0.21826171875, 0.28369140625, 1.1572265625, -0.402099609375, 1.935546875, 2.2734375, 1.0556640625, -1.443359375, 0.343505859375, -0.2353515625, -0.11138916015625, 0.11651611328125, -1.5068359375, -2.80078125, 0.7958984375, 2.546875, 1.0947265625, -0.25927734375, -1.06640625, 0.12420654296875, -2.04296875, -0.29833984375, 0.0250396728515625, -1.2841796875, 0.6884765625, -0.1982421875, 0.86767578125, 0.003448486328125, 2.564453125, 0.1474609375, -0.311279296875, 0.369384765625, -0.90966796875, -0.284423828125, 0.66796875, -0.71923828125, -0.41943359375, -1.546875, 0.77490234375, 1.0654296875, 0.052703857421875, -0.1041259765625, -2.212890625, 1.0126953125, 2.263671875, 0.984375, 0.10211181640625, 1.580078125, 1.2158203125, 1.173828125, -0.9501953125, -1.0576171875, -1.48828125, 1.6572265625, 0.07281494140625, 0.046234130859375, 0.5341796875, 1.015625, -0.6083984375, 1.041015625, 0.07861328125, 0.11529541015625, -1.76171875, 0.322265625, 2.197265625, -1.3017578125, 0.55029296875, -0.36474609375, 1.3095703125, -0.208984375, 1.138671875, -0.54052734375, 2.259765625, 1.17578125, -0.84130859375, 1.1328125, -2.0078125, 0.43115234375, 1.2626953125, -1.3720703125, 0.3330078125, -0.310302734375, 1.095703125, -0.473876953125, -0.66650390625, -0.09918212890625, 0.36865234375, -1.0390625, 0.75537109375, -0.61279296875, 0.34130859375, 0.9599609375, -0.8662109375, 0.50146484375, 0.2427978515625, -0.0830078125, 2.876953125, -0.136474609375, -0.482421875, -1.0751953125, 0.5048828125, -0.98974609375, 0.138916015625, 0.3671875, 1.3291015625, -0.7626953125, -0.30126953125, 1.33984375, 0.93798828125, 0.64794921875, 1.2392578125, 0.27685546875, 0.142578125, 0.47509765625, -2.19921875, 1.3056640625, -0.59326171875, -0.359619140625, -1.2509765625, 0.71630859375, 0.58740234375, -0.62646484375, -1.392578125, -2.37890625, -0.164306640625, 1.107421875, -0.11737060546875, 1.6494140625, 0.75146484375, 0.8779296875, -0.73486328125, -0.439208984375, 0.67626953125, 1.8212890625, -1.326171875, 0.72607421875, 1.1005859375, -0.337646484375, 2.177734375, 1.3076171875, -0.3466796875, -0.5087890625, 0.5791015625, -0.04229736328125, 0.83740234375, -2.01171875, -0.74658203125, 0.9287109375, 0.1614990234375, 0.21484375, 0.08038330078125, -0.2178955078125, 0.857421875, -0.354248046875, -1.294921875, 1.658203125, 0.2381591796875, -0.3857421875, -2.0078125, 0.1630859375, -1.4794921875, -2.1796875, -0.984375, -0.70947265625, 0.9453125, -1.3603515625, 0.1903076171875, -0.141357421875, 0.2215576171875, 0.1566162109375, -0.189208984375, -0.190673828125, 1.0810546875, 0.0164031982421875, -0.43115234375, -0.8564453125, 1.283203125, 0.409423828125, 0.1943359375, -1.2041015625, 0.2332763671875, 0.422607421875, 0.5888671875, 2.30078125, 1.099609375, 2.232421875, -0.6083984375, -0.262939453125, -0.1580810546875, -0.5, 0.337890625, -0.52978515625, -1.955078125, -0.4130859375, -0.347412109375, 2.099609375, 0.95166015625, -0.10455322265625, -1.978515625, 0.1580810546875, -1.8662109375, -0.035064697265625, -0.72216796875, 0.77880859375, 3.17578125, 0.67822265625, -0.479248046875, -1.7861328125, -0.7216796875, 0.054779052734375, -0.84423828125, 0.427490234375, 0.00794219970703125, -0.049285888671875, -0.9990234375, -0.89794921875, -0.1900634765625, 0.833984375, 2.982421875, 0.89599609375, 0.80517578125, 1.5361328125, 0.4755859375, -1.228515625, -0.433837890625, 0.29833984375, 0.70654296875, 1.4736328125, 0.75439453125, -0.8203125, -0.58984375, -0.07037353515625, 0.82421875, -0.56787109375, -0.23681640625, 0.39697265625, -0.6728515625, -0.56298828125, 0.1746826171875, -0.53271484375, -0.89208984375, 0.48095703125, 1.416015625, -0.55615234375, 0.0770263671875, 0.1685791015625, -0.488037109375, 0.84326171875, 1.3818359375, -0.483642578125, 0.771484375, -0.423095703125, 0.962890625, -1.6396484375, -1.0087890625, -1.19921875, -0.0188446044921875, 0.173583984375, 0.58349609375, 1.2998046875, -0.1263427734375, 0.21142578125, -0.80859375, -1.6767578125, -0.7666015625, -3.01953125, 1.2275390625, 0.85791015625, 0.48828125, -1.5, 0.427734375, 0.54345703125, -1.037109375, 0.0859375, -1.2041015625, -0.68359375, -0.345458984375, 1.3251953125, 0.13232421875, 1.009765625, -0.399169921875, -0.221435546875, 0.07159423828125, 0.387451171875, 0.390380859375, -0.462890625, -0.57177734375, -0.0631103515625, -0.48779296875, 1.193359375, -0.8212890625, 0.31396484375, -1.1474609375, 0.8623046875, 0.6240234375, 0.767578125, 0.275634765625, -1.2294921875, 2.318359375, 0.488525390625, 0.65087890625, 0.88330078125, -0.95947265625, -1.2939453125, -0.318359375, 1.0517578125, 0.77392578125, 0.264892578125, -0.436767578125, 0.3076171875, 1.33984375, 0.45703125, -0.10064697265625, -0.8828125, 0.4013671875, -0.323486328125, -0.7978515625, -0.254638671875, 0.297607421875, -0.2381591796875, 0.1201171875, -1.2529296875, -1.5126953125, -1.31640625, -0.003673553466796875, 0.65283203125, 2.00390625, -1.8388671875, 0.7119140625, 0.5478515625, -0.4287109375, 2.65234375, 0.1312255859375, 0.439697265625, 0.41015625, -0.78759765625, -1.7783203125, 0.047210693359375, -1.2841796875, 0.07403564453125, 1.3349609375, -1.4912109375, 2.435546875, -0.12396240234375, -1.4765625, 0.403564453125, -0.9931640625, -0.54052734375, 0.5146484375, 0.351318359375, 0.1400146484375, -0.375, 1.3408203125, 0.470947265625, -0.71533203125, 0.08843994140625, -0.66064453125, 0.30078125, -1.81640625, -0.01035308837890625, -0.04278564453125, 0.4521484375, -0.88037109375, -2.091796875, -0.7353515625, 1.4970703125, -0.0180511474609375, -0.5439453125, -0.032501220703125, -0.88037109375, 2.919921875, -0.755859375, -0.374267578125, -1.96875, -0.03350830078125, 0.469970703125, -1.2763671875, -0.307373046875, 1.40625, 0.287841796875, 0.9306640625, 0.89208984375, -1.41796875, 0.8212890625, -0.1956787109375, 1.8916015625, 0.0517578125, 0.49072265625, -2.517578125, -0.365234375, 0.6171875, 0.25927734375, -0.70166015625, -0.271728515625, -2.529296875, 1.998046875, -1.060546875, -0.391845703125, 1.3203125, 0.822265625, 1.8515625, 0.41796875, -0.0384521484375, -0.0634765625, -0.76953125, -0.382080078125, 0.650390625, -1.35546875, -0.453369140625, 0.7958984375, 0.86962890625, -0.09234619140625, 0.336669921875, -0.1292724609375, 0.384521484375, 0.59716796875, 1.447265625, -1.599609375, 0.69384765625, 0.908203125, -1.2099609375, -1.060546875, -0.9326171875, -1.2255859375, -2.796875, -1.572265625, 0.126220703125, -0.54248046875, -0.1580810546875, 0.38232421875, 1.2578125, 1.306640625, -2.251953125, 0.31396484375, 0.56103515625, 0.90771484375, -0.05267333984375, -0.12322998046875, -0.80517578125, 0.070068359375, -0.39306640625, 0.8408203125, -0.12115478515625, 0.0760498046875, 0.345458984375, -0.1636962890625, -0.045501708984375, -0.4619140625, 1.185546875, -2.197265625, 0.7666015625, 0.215576171875, 0.225830078125, 0.703125, -0.061767578125, -1.291015625, 0.274169921875, -0.96923828125, -0.419921875, -1.06640625, -1.4541015625, 1.8017578125, 1.3017578125, 1.0185546875, 0.290771484375, 0.3388671875, 1.6064453125, 0.284423828125, 0.67822265625, 1.4404296875, 0.3876953125, -1.2666015625, 0.18896484375, -0.93994140625, -0.0709228515625, -2.205078125, 1.3603515625, -1.3916015625, 0.41357421875, -2.01171875, 0.83447265625, -0.88525390625, -1.15234375, -1.4765625, 0.67333984375, -0.00390625, -0.7861328125, -0.728515625, 0.1171875, -1.443359375, -0.54052734375, -0.19873046875, -0.06951904296875, 0.264892578125, 1.501953125, 0.68701171875, -1.2041015625, 0.342041015625, -1.40625, 0.433349609375, 0.97705078125, 0.036956787109375, 0.9775390625, -0.16357421875, -0.1993408203125, 0.088623046875, 0.490234375, 0.11236572265625, 1.814453125, -1.5859375, -0.34375, 0.050079345703125, 0.367919921875, 0.441162109375, -0.10931396484375, 0.0526123046875, 1.3779296875, -0.205810546875, -1.513671875, 0.66552734375, -0.1456298828125, 0.2379150390625, -0.40478515625, -0.1656494140625, 0.62646484375, -0.1422119140625, -0.469482421875, 0.60986328125, 0.9306640625, 0.0272369384765625, 0.0455322265625, 0.75830078125, -0.82275390625, -0.57373046875, -1.2861328125, 2.330078125, 0.33642578125, 0.658203125, -1.2177734375, 0.62353515625, -1.7900390625, 0.83935546875, 0.8759765625, 0.08294677734375, -1.6474609375, -1.533203125, 1.6279296875, 0.478515625, -1.7060546875, -0.479248046875, 2.08203125, -0.41357421875, 0.74267578125, 0.869140625, 0.00305938720703125, -0.51708984375, -0.7529296875, 0.7080078125, -1.974609375, -0.9912109375, -0.12091064453125, -0.03753662109375, 0.29638671875, 0.283935546875, -1.330078125, -0.5458984375, -0.46337890625, -0.8623046875, 0.51318359375, -1.31640625, 0.1763916015625, -0.89697265625, -0.2120361328125, -0.443359375, 1.1171875, 0.38720703125, 0.01305389404296875, 0.0023040771484375, 1.3916015625, -0.1822509765625, 0.7275390625, -0.697265625, 1.845703125, -1.087890625, -0.8154296875, 1.5654296875, -0.2178955078125, -0.8896484375, -1.0546875, 1.220703125, 0.331298828125, -0.366455078125, 0.5302734375, -0.68798828125, -0.83056640625, -1.2578125, -0.150146484375, 1.5361328125, 2.35546875, -0.69091796875, 2.09375, 2.275390625, 0.481689453125, -0.60791015625, -0.6142578125, 0.6279296875, 0.85791015625, 1.25390625, 1.2041015625, -1.376953125, 0.12493896484375, 0.2161865234375, -0.2435302734375, 0.93505859375, 1.2197265625, -0.25830078125, 1.05859375, 0.248779296875, 1.07421875, -1.5400390625, 1.5439453125, 1.4013671875, -1.046875, 0.5986328125, 0.489501953125, 0.21240234375, -1.2734375, 0.6884765625, 0.55078125, -0.80224609375, -1.64453125, -0.1351318359375, -0.069091796875, -1.494140625, -1.1337890625, 0.998046875, 0.1644287109375, -0.01262664794921875, 0.77783203125, 0.30615234375, 1.021484375, -0.83935546875, -0.46044921875, 1.0048828125, 0.453125, 1.578125, -1.236328125, 0.63525390625, 0.004405975341796875, -0.370849609375, 1.3134765625, -2.048828125, -0.51806640625, 0.204833984375, 0.009063720703125, -0.436767578125, -0.3515625, 0.351318359375, -0.7080078125, -0.1925048828125, -1.1123046875, -0.76171875, 3.05859375, 0.2425537109375, 2.796875, -1.583984375, -1.6298828125, 0.383056640625, -1.2099609375, -0.296630859375, -0.7578125, 0.2213134765625, 0.0001208186149597168, -0.658203125, -1.8447265625, 1.689453125, -1.30859375, 1.6962890625, -0.3818359375, -0.548828125, 0.6435546875, -0.2005615234375, -2.01953125, 0.141357421875, 0.302978515625, -0.76904296875, 0.054107666015625, 1.1259765625, -0.4072265625, 0.1781005859375, -0.22509765625, 0.173583984375, 1.986328125, 0.97607421875, 1.4365234375, 0.80517578125, -1.2177734375, 1.7822265625, 0.482421875, 0.90234375, 0.320556640625, 0.68408203125, 0.763671875, 1.64453125, 1.19140625, 0.8876953125, 0.46533203125, 0.256103515625, 1.6962890625, 0.05413818359375, -0.00839996337890625, 0.6435546875, -0.3828125, 1.0107421875, 1.5634765625, -0.6337890625, -0.78369140625, 1.236328125, -1.47265625, 0.59716796875, 0.58984375, -1.01171875, 1.7373046875, -0.1513671875, 0.497802734375, -1.4384765625, 0.351318359375, -0.20947265625, -0.31005859375, 0.6396484375, 0.20849609375, 0.83154296875, -1.1640625, 1.349609375, -1.05078125, -1.462890625, 1.5771484375, 0.399169921875, -1.244140625, 0.63720703125, 0.0799560546875, 0.259521484375, 0.310302734375, -0.68896484375, -0.87060546875, -0.58203125, -0.0537109375, 1.2646484375, 0.484619140625, 1.767578125, 0.2069091796875, -0.1412353515625, 0.9501953125, 0.943359375, -0.529296875, -0.54736328125, -1.4501953125, -0.44873046875, 1.0341796875, 1.7998046875, -0.515625, 0.572265625, -0.73046875, 2.130859375, -1.3330078125, -0.035400390625, 1.2978515625, -1.1103515625, 0.71923828125, 1.423828125, -2.091796875, 0.765625, -1.390625, 0.17822265625, 1.1220703125, 1.1552734375, -0.09002685546875, 0.7744140625, 1.9658203125, -0.8330078125, 0.422607421875, 0.490966796875, -0.221435546875, 0.87158203125, 1.2294921875, 0.3583984375, 1.1181640625, -0.65234375, 1.78515625, -0.107177734375, 0.92919921875, -1.26953125, 3.52734375, 0.08380126953125, 0.2125244140625, -0.27734375, -0.90771484375, 0.7412109375, 0.1947021484375, 1.4873046875, 0.216796875, -0.06524658203125, 0.95654296875, -0.5732421875, 2.8125, 1.6474609375, 0.3779296875, 0.54296875, 0.89794921875, -0.383056640625, -0.54443359375, 1.4208984375, -0.408203125, -1.146484375, -0.0858154296875, 1.625, -1.2685546875, -0.8662109375, 1.5576171875, 1.12109375, -1.1015625, -0.55029296875, 1.650390625, 0.75146484375, -1.18359375, 0.25390625, 1.02734375, -1.3544921875, 0.53564453125, -1.6494140625, -1.623046875, 0.29833984375, 2.986328125, 2.30078125, 0.822265625, 0.458251953125, 1.3583984375, -0.2415771484375, -0.379150390625, -0.4326171875, 0.306640625, 0.0360107421875, 0.4658203125, 0.7578125, 1.6943359375, 1.3955078125, -0.49755859375, 1.3525390625, 1.4345703125, 0.022552490234375, 0.537109375, 0.11578369140625, -1.3974609375, 0.4912109375, 0.5205078125, 1.23828125, 0.492919921875, 0.3271484375, 0.150390625, -0.315185546875, 0.202880859375, -1.3623046875, -0.81591796875, -1.0498046875, -0.0283203125, 0.8759765625, -0.447021484375, 0.2342529296875, 0.128662109375, 0.035125732421875, 0.61474609375, -1.703125, 0.71484375, -0.62939453125, -0.12384033203125, -0.1427001953125, -2.1171875, -0.560546875, -0.1370849609375, 0.7138671875, -0.50927734375, -1.0576171875, -0.0887451171875, -0.77099609375, 0.471435546875, -0.3447265625, 0.69189453125, 0.301513671875, -0.0780029296875, 0.767578125, -0.86474609375, -0.4267578125, 1.0166015625, 0.3720703125, -0.5556640625, 0.77685546875, -1.4384765625, 0.560546875, -0.28662109375, -1.818359375, 0.10992431640625, -1.5546875, -1.1904296875, 1.240234375, 1.4111328125, -0.047607421875, 0.24658203125, -0.4443359375, 0.0914306640625, 0.85009765625, 0.51318359375, -0.767578125, -0.281494140625, 0.275634765625, -0.56689453125, 0.453125, 1.2021484375, -0.9267578125, 0.36376953125, -0.4052734375, -1.216796875, -0.45458984375, -0.06903076171875, -0.76806640625, -0.6650390625, 1.115234375, 1.8916015625, -0.62353515625, -0.6142578125, 0.8017578125, -2.26171875, 1.205078125, -0.7333984375, -0.99169921875, -0.06927490234375, -0.300537109375, 0.587890625, -1.4755859375, -0.454833984375, 0.54443359375, 0.791015625, 0.70947265625, 0.53955078125, -1.794921875, -0.6943359375, -0.250244140625, 1.44921875, 0.888671875, 0.263427734375, -1.494140625, -0.548828125, 0.80029296875, -0.51171875, 0.314697265625, 0.145751953125, 0.52099609375, -0.88818359375, -0.24462890625, -1.5234375, -0.262939453125, -1.1748046875, -1.8046875, -0.14306640625, 0.468505859375, -0.11505126953125, 0.138916015625, 1.240234375, 0.11859130859375, -1.078125, 1.3779296875, 0.53515625, -0.227294921875, -0.52099609375, 1.8017578125, 1.2685546875, -2.08984375, 0.04266357421875, -0.35107421875, -0.685546875, 0.892578125, 0.095703125, 0.869140625, -0.759765625, -1.7783203125, -0.443115234375, 0.09442138671875, -0.454345703125, 0.970703125, 0.53466796875, -0.6875, 0.314208984375, -0.2685546875, 1.6435546875, 1.015625, -0.06768798828125, -0.39794921875, 0.064453125, -0.367919921875, -0.35546875, -0.1038818359375, 1.7470703125, -0.109375, 2.025390625, 0.150146484375, -0.0439453125, -1.2275390625, 1.8798828125, -1.173828125, -1.208984375, -0.84716796875, -1.80078125, -0.958984375, -0.3828125, 1.529296875, 1.8212890625, -0.2646484375, 0.01398468017578125, -0.3125, -1.4814453125, 0.32861328125, 0.76171875, 0.2236328125, -0.89599609375, 0.68017578125, -0.207763671875, 0.341552734375, 0.1468505859375, -1.0234375, 1.0048828125, 0.197998046875, 0.276611328125, 0.23779296875, 0.27978515625, 1.1552734375, 0.299072265625, -1.0537109375, 0.0478515625, 1.009765625, -0.51416015625, -0.0099334716796875, -0.224609375, 0.57275390625, -0.94775390625, 1.1416015625, 0.8984375, 0.87939453125, -0.77490234375, -0.90478515625, -0.279052734375, 1.060546875, 0.34912109375, -0.603515625, -1.0908203125, -0.353759765625, -0.282470703125, 0.038055419921875, 0.1402587890625, -1.810546875, -0.176025390625, -0.76904296875, 1.0615234375, -1.9189453125, 0.95947265625, 0.39501953125, -0.71240234375, -1.673828125, -0.61865234375, 0.400146484375, 0.01300048828125, 0.55712890625, -0.0625, 0.56689453125, 0.2802734375, -0.423095703125, 0.8408203125, 0.85546875, -2.5234375, -0.3447265625, -1.26171875, 1.208984375, 0.5556640625, 0.7421875, -0.7431640625, 0.0843505859375, -0.99267578125, 0.25830078125, 0.57666015625, -0.2130126953125, 0.07171630859375, -0.4150390625, 0.919921875, 0.93310546875, -0.5107421875, -0.8447265625, 1.75390625, -0.2171630859375, -0.0986328125, -0.471923828125, 1.236328125, 0.9306640625, -0.68115234375, -1.18359375, 0.4453125, 0.270263671875, 0.61865234375, 0.6064453125, -1.134765625, 0.309814453125, -1.0, -0.76611328125, 0.87841796875, 2.103515625, -0.332763671875, -1.5546875, -0.58154296875, 0.7509765625, 1.42578125, 0.357421875, -1.83203125, 0.12890625, 0.88671875, 0.009002685546875, 1.017578125, 0.88525390625, -0.1517333984375, 0.00904083251953125, -0.419189453125, -1.6259765625, 1.59375, -0.1529541015625, 0.140380859375, 0.1783447265625, -0.8427734375, -1.6083984375, 0.1966552734375, 1.2998046875, -1.5986328125, -0.5927734375, -0.433837890625, -0.79443359375, 1.982421875, -0.75537109375, -1.2919921875, -0.25341796875, -2.048828125, -1.4794921875, -1.4990234375, -0.474853515625, -0.183837890625, 0.1480712890625, 0.06805419921875, 0.8388671875, 0.033172607421875, 0.435546875, 0.345458984375, -1.2900390625, 0.72802734375, 0.671875, 0.0679931640625, -0.57568359375, -2.0078125, 0.1416015625, -0.54296875, -0.491943359375, 0.84912109375, 0.1961669921875, 2.00390625, -0.08831787109375, 1.515625, -0.74267578125, -1.400390625, 1.2333984375, 1.0107421875, 0.452392578125, 0.71142578125, -1.314453125, -1.2685546875, -1.3271484375, 1.0947265625, 0.5986328125, 0.277099609375, 1.1318359375, 0.057037353515625, 1.0400390625, 1.2939453125, -0.0711669921875, 0.53076171875, -1.3759765625, 0.16650390625, 0.6328125, 1.228515625, -0.95751953125, -0.337158203125, 2.2578125, 0.77294921875, -0.5087890625, -0.0858154296875, 0.45556640625, 2.1328125, -0.92822265625, 0.72802734375, -0.71044921875, -0.6015625, 1.7490234375, -0.8154296875, 0.56298828125, -0.962890625, 0.3017578125, 0.79833984375, -0.2081298828125, 0.79150390625, 0.86865234375, 1.3193359375, 2.21875, -0.00951385498046875, 0.1220703125, -1.76171875, -0.26220703125, -1.494140625, 0.83984375, 0.93701171875, -2.689453125, 0.9599609375, 0.01306915283203125, 1.9716796875, -0.974609375, -1.1591796875, -1.361328125, 0.039276123046875, 0.1925048828125, -0.72607421875, 0.321044921875, -1.48828125, -2.119140625, 1.259765625, 0.97216796875, 1.724609375, -0.61083984375, -0.414306640625, -0.222412109375, 0.919921875, -0.84619140625, -0.9521484375, -0.10760498046875, 1.1123046875, 0.380859375, -0.02301025390625, -1.9560546875, 1.0400390625, 0.0250396728515625, 0.46728515625, -1.3798828125, 0.279296875, -0.142333984375, -1.2275390625, 0.77880859375, 0.232421875, -1.3525390625, -0.55859375, 0.01873779296875, 0.451171875, -0.06988525390625, 0.564453125, 0.417236328125, -1.55078125, 1.18359375, -1.1142578125, 0.5830078125, -1.880859375, 0.042572021484375, -1.5419921875, 1.1298828125, -1.107421875, -0.6337890625, 1.1376953125, 0.76611328125, 0.1893310546875, -0.37158203125, 0.1036376953125, -0.469482421875, 0.677734375, 1.2607421875, 2.041015625, 0.0728759765625, 1.9765625, -0.654296875, 0.36767578125, 0.11053466796875, 0.99267578125, -2.451171875, -1.353515625, -0.46484375, 2.080078125, 0.55859375, -0.27587890625, 1.5712890625, -0.62158203125, -0.5458984375, 0.07781982421875, -1.603515625, -1.4462890625, -1.3447265625, -0.58642578125, 0.35498046875, 1.4482421875, 0.1953125, -0.90283203125, 2.181640625, -1.142578125, 0.1893310546875, 1.0771484375, 0.08428955078125, -0.23779296875, 0.59521484375, -0.096435546875, -0.5224609375, 0.73876953125, 0.3994140625, 1.0478515625, -0.039581298828125, -1.375, 0.0335693359375, -0.55712890625, 1.6708984375, -0.88330078125, -1.0625, 1.0205078125, 0.95751953125, 0.8837890625, 1.36328125, 0.64306640625, 0.7060546875, 0.8603515625, 0.79248046875, -0.78369140625, -0.69970703125, 1.9677734375, -1.765625, 1.240234375, -1.125, 2.23828125, 0.03594970703125, 0.24609375, 0.1380615234375, 0.65478515625, -0.0240631103515625, 0.3115234375, 0.1981201171875, 0.6953125, -0.3720703125, 0.281005859375, -1.01953125, 0.459716796875, 2.279296875, -2.115234375, 1.939453125, -0.78955078125, -0.49853515625, 0.466064453125, -0.7060546875, 1.75, 1.7734375, -0.9921875, 0.6123046875, -0.79833984375, 1.0185546875, 1.03515625, 0.2232666015625, -0.666015625, -2.1953125, 0.49609375, -0.96337890625, 0.0008859634399414062, -1.025390625, 0.1654052734375, -0.07257080078125, 1.8701171875, 1.224609375, -0.07867431640625, -0.425048828125, 0.243896484375, 0.5712890625, 1.9287109375, -1.01171875, -0.591796875, -1.0048828125, -1.34375, 0.0004940032958984375, 0.03656005859375, -1.466796875, -0.58544921875, -0.77001953125, -1.265625, -1.2470703125, -0.2100830078125, -1.7021484375, -0.91455078125, 0.83056640625, -1.65625, -0.041015625, -0.008087158203125, 0.53466796875, -0.54833984375, -0.426025390625, -2.02734375, -0.373046875, -1.8759765625, -1.841796875, 1.126953125, 0.6533203125, 0.42822265625, 0.489501953125, -0.42822265625, 0.463623046875, 0.662109375, -1.1005859375, -0.271240234375, -0.615234375, 0.01300811767578125, 0.9306640625, 1.51171875, -1.83984375, 0.325439453125, 0.144775390625, -0.1182861328125, 1.21484375, 1.669921875, -0.8447265625, 0.04144287109375, 1.3408203125, 0.058502197265625, 0.64794921875, -0.281494140625, 0.408203125, -1.9248046875, 0.290283203125, 0.091552734375, -1.07421875, -1.7587890625, 1.515625, 1.1455078125, 0.28466796875, -0.72412109375, 0.305419921875, 0.427490234375, 0.347412109375, 2.71875, -1.583984375, 0.62548828125, -0.12384033203125, -0.54345703125, 0.73486328125, 0.024200439453125, -0.074951171875, -1.087890625, 0.25341796875, -1.6611328125, 0.54052734375, 0.193115234375, -1.5439453125, -1.701171875, 1.6669921875, 0.6943359375, 2.044921875, -1.7568359375, 0.10406494140625, 0.11151123046875, 0.184326171875, 0.313232421875, 1.6025390625, -1.6083984375, 0.70068359375, -2.47265625, -0.5400390625, -0.935546875, -2.404296875, 0.71630859375, -0.9609375, -0.97021484375, 0.8017578125, 1.748046875, 0.533203125, -1.029296875, -1.4775390625, -1.248046875, 0.330078125, -0.88818359375, -1.0859375, -0.439208984375, 0.8271484375, -0.55517578125, -0.050384521484375, 0.440673828125, -0.1478271484375, 0.5185546875, -0.734375, 0.271728515625, -0.76171875, 0.3310546875, 0.0054473876953125, -0.47705078125, 0.22021484375, -0.880859375, -0.0094451904296875, -0.2763671875, -1.0634765625, 1.115234375, -1.4736328125, 0.08416748046875, 1.267578125, 0.08221435546875, -0.87109375, -0.595703125, 1.087890625, 1.8037109375, 0.99462890625, 0.85791015625, 2.01171875, -0.59423828125, -1.9091796875, 0.7822265625, -1.4765625, 0.736328125, -0.440673828125, -0.3564453125, -1.169921875, 0.161865234375, 0.88525390625, 1.4384765625, -0.8544921875, -1.7646484375, 0.40087890625, -0.1971435546875, 0.52294921875, -0.6220703125, -1.841796875, -1.294921875, 0.0413818359375, 0.91845703125, 1.06640625, 0.5576171875, -0.837890625, 0.9228515625, -0.1378173828125, 1.361328125, -0.5986328125, 1.2509765625, -1.3515625, -1.150390625, 1.0234375, 1.3671875, 0.296630859375, 1.1298828125, -0.1868896484375, 0.296630859375, -0.5615234375, 0.6474609375, -0.2083740234375, -1.8125, 0.164794921875, -0.491943359375, 0.01334381103515625, 1.193359375, -1.0400390625, 0.7958984375, -0.77099609375, -0.70263671875, -1.923828125, -0.72900390625, 0.07379150390625, 0.66064453125, 0.079345703125, -1.5703125, -0.9990234375, 0.4384765625, 1.79296875, -1.083984375, -0.99560546875, -0.74755859375, -1.6435546875, 0.75244140625, 0.1260986328125, 0.12890625, 0.386474609375, 1.15234375, -1.734375, -0.55029296875, 0.406982421875, 1.4248046875, -1.396484375, 0.76904296875, -0.223388671875, -1.1298828125, 0.266845703125, -0.09844970703125, 1.287109375, -0.408203125, -1.783203125, -0.50927734375, -0.626953125, -1.015625, 0.3671875, -0.1298828125, -1.421875, 0.75244140625, -0.442626953125, -0.0031566619873046875, 0.1641845703125, -0.434814453125, 2.02734375, -0.8427734375, 0.0791015625, 0.50732421875, 1.87890625, 1.615234375, 0.74755859375, 0.77734375, -0.466552734375, 0.30517578125, -0.7841796875, -4.15234375, -1.060546875, -0.09710693359375, -0.91552734375, -0.8154296875, -0.2078857421875, 0.450927734375, 0.517578125, -0.2333984375, 0.2152099609375, 0.1973876953125, 0.76123046875, -0.1820068359375, 0.65234375, 1.1796875, 0.73876953125, -0.04864501953125, -3.04296875, -1.6142578125, -0.7197265625, 1.0009765625, -2.35546875, 0.459228515625, -2.361328125, 0.43408203125, -1.7451171875, 0.443603515625, -0.9560546875, 1.2568359375, -0.65966796875, -0.796875, -0.8349609375, 0.7021484375, -0.1636962890625, 0.57568359375, -0.2457275390625, 0.424072265625, -1.666015625, 0.4453125, -0.3857421875, -0.83544921875, 0.771484375, -1.26953125, -0.8291015625, -0.82177734375, -0.56689453125, -0.40673828125, -0.410888671875, 0.96826171875, -1.005859375, -0.327392578125, 0.86962890625, 0.15771484375, 1.4052734375, -0.93505859375, -0.3564453125, 0.291748046875, 0.7080078125, 1.6845703125, 0.1859130859375, -1.46875, 0.2177734375, -0.463623046875, 0.8818359375, 1.138671875, 0.50048828125, 2.654296875, 2.220703125, 1.1591796875, -0.94091796875, 1.271484375, -0.982421875, 0.90234375, 0.28515625, -0.08978271484375, -1.0322265625, 0.5390625, 0.52734375, -1.0498046875, 0.402587890625, -0.05889892578125, 0.6201171875, 0.1446533203125, -0.5732421875, -0.1424560546875, -0.51513671875, -0.74072265625, -0.13818359375, 0.1278076171875, 0.70458984375, -1.015625, -0.08599853515625, 0.6171875, -0.35595703125, 0.658203125, 0.06671142578125, -0.498046875, -0.035614013671875, 1.150390625, 0.09185791015625, 0.65625, 0.1474609375, -0.209228515625, -0.869140625, -0.30859375, -0.33349609375, 0.70751953125, 1.8896484375, 0.1727294921875, -1.044921875, -1.3740234375, -1.00390625, 0.0127716064453125, 1.3662109375, -2.5625, 0.56201171875, -1.3125, -1.380859375, -1.5439453125, 0.2269287109375, 0.4716796875, 0.46337890625, -1.072265625, 0.52490234375, -0.40771484375, -1.390625, 0.249267578125, 0.469482421875, 0.375732421875, -0.611328125, 0.440673828125, -1.8955078125, 0.50927734375, -0.41845703125, 1.791015625, -0.287841796875, 0.397216796875, -0.0200958251953125, 1.234375, -0.256591796875, -0.2342529296875, -0.1978759765625, 0.90673828125, 0.4150390625, 0.29541015625, 1.4296875, 0.380615234375, -0.56884765625, -1.3125, 0.73974609375, 0.283935546875, 2.056640625, 0.2763671875, 1.09375, -0.1356201171875, 0.295654296875, 0.09918212890625, -0.369873046875, -1.6064453125, -0.91943359375, -1.330078125, -0.17626953125, -0.87548828125, 0.8603515625, -0.062469482421875, 0.619140625, -0.73828125, 0.492431640625, 0.294921875, 0.63232421875, 0.336181640625, 1.8486328125, -0.58056640625, -0.998046875, 0.98291015625, -0.00981903076171875, -1.14453125, 1.0, 0.4990234375, -1.3154296875, 0.65966796875, 0.87744140625, -0.384033203125, 0.59423828125, 1.3662109375, 0.1864013671875, -0.085693359375, -0.76611328125, 1.3828125, 1.8916015625, -0.6416015625, -1.85546875, 1.05078125, -0.0234832763671875, -0.52587890625, -1.0615234375, 0.468994140625, 1.5771484375, 0.5859375, -0.64697265625, -0.375244140625, 0.041595458984375, -1.0908203125, 0.2783203125, 0.343505859375, 1.0693359375, 1.408203125, -0.51513671875, 1.2548828125, -0.17724609375, -0.25830078125, -0.72802734375, 1.1142578125, 0.07952880859375, -0.0284881591796875, -0.48681640625, 2.00390625, 0.66796875, -0.9296875, 0.29345703125, -0.08843994140625, -1.1474609375, -1.029296875, -0.96044921875, -0.7353515625, 1.1455078125, 0.25146484375, -0.39697265625, -1.703125, -0.73291015625, 2.220703125, -0.060546875, 0.058624267578125, -1.1865234375, -1.6455078125, 1.0986328125, 0.57177734375, 1.6689453125, -0.54833984375, -0.1484375, -0.62646484375, 0.0232086181640625, -0.81591796875, 0.8818359375, -0.64794921875, 0.7890625, 0.63525390625, -0.3349609375, 0.057037353515625, 0.03216552734375, -0.39208984375, -2.111328125, 1.4658203125, 0.408935546875, -0.7041015625, 1.05859375, 2.029296875, -0.97705078125, 1.1044921875, -0.12451171875, -0.8818359375, 0.19775390625, 0.80615234375, -0.16064453125, 0.31689453125, -1.0283203125, 0.41357421875, 1.0615234375, 1.505859375, 0.40234375, 0.303955078125, 1.3603515625, -0.72607421875, -1.39453125, -0.0174102783203125, -0.7783203125, -2.140625, 1.1455078125, -1.1650390625, 1.5673828125, 0.71484375, -0.4208984375, -0.9296875, 0.004291534423828125, -0.67919921875, 1.3388671875, 0.1220703125, 0.299560546875, -0.036956787109375, -0.79638671875, -1.5341796875, 1.328125, 0.8291015625, 0.415283203125, 0.342041015625, 0.66796875, 0.72509765625, -1.4013671875, -0.0579833984375, 0.9072265625, -0.453369140625, 0.9111328125, -0.2939453125, 0.3603515625, 0.501953125, -1.9560546875, -1.4287109375, 0.364990234375, 1.2001953125, -1.4453125, -0.64892578125, -0.2196044921875, 1.1796875, -0.11309814453125, 0.01354217529296875, 1.3115234375, -0.8056640625, -1.54296875, -0.49365234375, -0.261474609375, 1.6376953125, 0.70166015625, -1.0693359375, 0.21533203125, 0.311279296875, -0.265869140625, -0.9345703125, 0.84716796875, 1.048828125, 0.115478515625, -1.0048828125, -0.83349609375, -0.1534423828125, -0.6845703125, 0.9560546875, -0.10791015625, -0.4931640625, 1.181640625, 0.70751953125, -0.27197265625, -1.66015625, -0.46875, -0.736328125, 1.0966796875, 1.0048828125, 1.0966796875, 0.06927490234375, 0.456787109375, 1.5537109375, -0.310546875, -0.77734375, -1.1474609375, -0.333251953125, -1.0048828125, -0.199462890625, -0.91357421875, 0.6337890625, 0.517578125, -1.265625, 1.53515625, 0.9873046875, -0.87060546875, 0.5625, -0.4765625, 0.52490234375, -1.5888671875, 0.222412109375, -1.099609375, -0.845703125, -0.32763671875, 0.2481689453125, 2.837890625, -0.226806640625, -0.2236328125, -1.4072265625, -1.9736328125, 0.5927734375, -0.81787109375, -0.0289764404296875, 0.1650390625, -1.4658203125, -0.53076171875, -0.040985107421875, -2.064453125, -0.61083984375, -1.1875, -1.3125, 1.6259765625, 0.336669921875, 0.423583984375, -2.001953125, 1.0322265625, 0.409423828125, 0.1993408203125, -0.187744140625, -0.10791015625, 0.45947265625, -1.185546875, 0.65673828125, -0.580078125, 1.13671875, -1.498046875, -0.15625, -0.2452392578125, 1.1455078125, 0.483642578125, 0.79736328125, -0.24609375, -0.212890625, 0.068359375, -0.75, -0.0011358261108398438, 0.285400390625, 0.08636474609375, 1.767578125, -0.70166015625, -0.057708740234375, 0.409912109375, -1.6240234375, -0.61669921875, 1.1572265625, -1.4599609375, 0.2349853515625, -0.591796875, 0.0172882080078125, 0.126708984375, 1.388671875, -0.970703125, -0.85107421875, -0.331787109375, 2.546875, 1.01953125, -1.8828125, -1.72265625, 0.673828125, -0.38427734375, -0.23046875, -0.60986328125, 0.716796875, 1.1728515625, -1.4677734375, -0.9736328125, -0.491943359375, 1.6279296875, 0.78125, 0.79248046875, 1.05078125, 1.833984375, 0.80029296875, -0.001312255859375, -0.96240234375, 0.14599609375, -0.237548828125, -0.6533203125, 0.0606689453125, 0.408935546875, 0.7314453125, -1.9375, 0.994140625, 1.2412109375, -0.193115234375, -1.083984375, -1.4892578125, 0.1405029296875, 0.595703125, -0.2685546875, -0.11669921875, 0.44287109375, -0.040374755859375, -0.379150390625, -0.9951171875, 1.447265625, 0.2337646484375, 0.5771484375, -0.01328277587890625, 0.8349609375, -1.3310546875, 1.01953125, -0.1083984375, -1.482421875, 2.0625, 0.09033203125, 0.1627197265625, 0.33642578125, 0.6591796875, -0.64208984375, 1.7880859375, -0.1588134765625, -0.4658203125, -0.0185546875, 0.419677734375, 0.78759765625, -0.7236328125, 0.200439453125, -0.2449951171875, -0.447998046875, 0.2724609375, 0.96875, -0.485107421875, -0.286865234375, 0.17578125, -0.4501953125, 0.371337890625, 0.6044921875, 0.0214996337890625, -0.243896484375, 0.445556640625, 0.64306640625, 2.6328125, 0.09490966796875, 0.00933074951171875, 0.59228515625, -2.07421875, -0.042205810546875, 0.826171875, 3.728515625, -1.5107421875, 0.8408203125, -0.86767578125, -0.70947265625, -0.0254669189453125, 0.14697265625, -0.06304931640625, -1.7578125, 0.9560546875, 0.1588134765625, -1.05859375, 0.7470703125, 0.62939453125, -1.37109375, -0.76220703125, 0.424560546875, -0.87939453125, -0.2125244140625, -0.84423828125, -0.7353515625, 0.1318359375, 0.2027587890625, -2.55859375, 0.297119140625, 1.12890625, 1.1484375, 0.350830078125, 0.037689208984375, -0.330322265625, 2.462890625, 1.6787109375, -2.15625, 0.115234375, -0.62109375, -0.26953125, -1.03515625, 1.109375, -1.0908203125, -0.400146484375, 0.415283203125, 0.68505859375, 0.196044921875, 0.92578125, 0.7314453125, -0.849609375, 0.603515625, -0.1341552734375, 0.8076171875, -1.2998046875, 0.07373046875, -1.1171875, 0.0222930908203125, 1.751953125, -1.4091796875, 0.66015625, 0.76611328125, 0.34130859375, -0.0288543701171875, -0.0143280029296875, 0.59033203125, 1.2685546875, 0.302490234375, -0.52490234375, 0.7802734375, -1.1591796875, 1.4033203125, -2.009765625, 0.51904296875, 0.432373046875, -0.93603515625, -0.44140625, 0.10382080078125, -0.86962890625, 1.1162109375, 0.533203125, -1.9501953125, 0.5126953125, 0.1505126953125, 0.281005859375, -0.4853515625, -1.791015625, 0.054962158203125, 0.921875, 1.0341796875, 0.8759765625, 0.25537109375, 0.4892578125, -0.0806884765625, 0.39013671875, 1.6455078125, 0.1395263671875, -0.6484375, -0.010009765625, 0.50634765625, 0.8837890625, 0.61083984375, -0.1241455078125, -0.658203125, -0.2236328125, 1.0546875, 0.0286712646484375, -1.40625, -2.0546875, 0.79296875, 0.15966796875, -1.0234375, -0.5615234375, 1.3671875, 0.6337890625, -0.0025997161865234375, 0.828125, -0.273193359375, 0.356689453125, -1.45703125, -0.65771484375, 2.078125, 0.4453125, -0.073486328125, 0.2288818359375, -0.049774169921875, 0.94140625, -0.0894775390625, 0.1600341796875, 0.8720703125, -0.479736328125, 1.3349609375, 0.273193359375, 1.8798828125, 0.8828125, -0.2342529296875, 0.77880859375, -0.20361328125, 1.0185546875, -0.63525390625, -1.849609375, 0.72216796875, 1.2841796875, 0.1866455078125, -1.0791015625, -1.5556640625, 0.125732421875, -1.8330078125, 1.109375, 0.4697265625, 0.7587890625, 0.69091796875, -0.54638671875, 1.115234375, -0.12396240234375, -0.05242919921875, 0.8544921875, 1.0556640625, -1.5498046875, -0.21142578125, 0.97900390625, -0.1318359375, 0.03302001953125, -1.1416015625, -0.163818359375, 0.11541748046875, -0.410888671875, 0.10162353515625, 2.40234375, 1.7001953125, -0.09893798828125, -1.2822265625, 0.036529541015625, -0.0855712890625, 1.515625, -1.21484375, 1.1943359375, -2.1015625, 0.759765625, -0.466552734375, 2.21875, 1.15625, 0.85400390625, -0.2437744140625, -0.4638671875, 0.70263671875, -0.8251953125, 0.60009765625, 0.2269287109375, -0.134033203125, -1.6376953125, 0.4384765625, -0.20654296875, -0.2486572265625, 0.09320068359375, 1.0009765625, 0.33984375, 1.515625, 0.415771484375, -0.09942626953125, 0.268310546875, -0.3095703125, 0.0859375, -0.87353515625, 0.407958984375, -0.7607421875, 0.10321044921875, -2.2578125, 0.63330078125, -1.2255859375, 1.0751953125, -1.19140625, 1.1044921875, 0.1956787109375, 0.76123046875, -0.96337890625, 2.287109375, 1.33984375, 1.2919921875, 0.0338134765625, 0.358642578125, -0.7255859375, -1.4560546875, -0.8369140625, 0.84326171875, 0.036224365234375, -1.21875, 1.1533203125, -0.338134765625, 1.201171875, -1.103515625, 1.5791015625, -0.50390625, -0.314208984375, -0.338623046875, -0.95068359375, 0.88916015625, -2.14453125, -0.0550537109375, -1.16796875, -1.5224609375, 5.5789947509765625e-05, 0.328857421875, -1.259765625, 0.424072265625, 1.1142578125, -0.72900390625, -1.3583984375, 0.86279296875, 0.68408203125, -2.1171875, -0.875, 2.310546875, 0.5498046875, -1.0966796875, 0.5751953125, 0.6416015625, -1.12109375, 0.806640625, 0.96923828125, -0.498291015625, -0.7802734375, 1.7470703125, 0.97314453125, -0.595703125, -0.65966796875, 0.46728515625, 1.1630859375, 1.08984375, 0.029632568359375, 0.5009765625, 0.081787109375, 1.0693359375, 0.82568359375, -0.63134765625, -0.71728515625, -0.9111328125, -1.31640625, 0.58154296875, 0.07830810546875, 0.92919921875, -0.49755859375, -1.02734375, 0.86767578125, 0.9228515625, -0.4501953125, -0.68701171875, -1.0185546875, -1.91796875, -0.73095703125, 0.400146484375, 1.115234375, -0.09686279296875, -0.60595703125, -1.3173828125, -0.9091796875, 0.6171875, -0.4052734375, 0.39892578125, -0.230712890625, -0.2415771484375, -0.056884765625, 0.197265625, -0.36474609375, 0.95751953125, 1.0224609375, -0.62548828125, 0.80029296875, -0.3984375, 0.287841796875, 1.0419921875, -0.277099609375, -0.8076171875, -0.96875, 1.3193359375, 1.2509765625, 0.27978515625, -0.65234375, -0.52197265625, 0.1181640625, 0.1279296875, -0.44580078125, 0.8232421875, 0.10009765625, 2.154296875, 0.5771484375, 0.77734375, 1.962890625, -0.58056640625, 0.142822265625, -0.56103515625, 1.572265625, 1.255859375, 0.861328125, -0.340087890625, 0.65625, -0.5126953125, 0.77197265625, -0.33447265625, 0.919921875, -1.4189453125, 0.34228515625, 1.677734375, -0.23291015625, -0.0032024383544921875, 0.40234375, 0.71923828125, -0.69140625, -1.921875, -1.3818359375, -1.666015625, 0.5419921875, 0.024749755859375, 0.9794921875, -1.322265625, 0.31494140625, -0.6943359375, 0.47509765625, -0.642578125, -0.70849609375, -0.28515625, 1.2060546875, 0.30517578125, 1.060546875, -2.275390625, -0.322021484375, -1.9638671875, -2.1484375, 1.17578125, -0.460205078125, 0.45263671875, -0.0758056640625, -0.96923828125, 0.0758056640625, -0.61181640625, 0.88037109375, -0.2159423828125, -0.049224853515625, 0.33349609375, -0.58056640625, -0.44287109375, 0.91845703125, 1.4892578125, -0.404541015625, -0.140625, -0.9853515625, 1.6044921875, 0.65625, 1.7158203125, 0.340576171875, -0.9052734375, -0.135986328125, 1.1591796875, -1.55078125, -1.2421875, -0.68017578125, 0.87939453125, -1.00390625, 1.3134765625, 0.40625, -2.14453125, 0.273681640625, -0.712890625, 0.43896484375, -0.59521484375, 0.2283935546875, 1.2431640625, 0.0791015625, -2.25390625, 0.2200927734375, -0.01025390625, 1.4287109375, -0.78955078125, 0.1080322265625, -0.307373046875, -1.169921875, -0.364013671875, -0.9697265625, 1.94921875, -0.049530029296875, -1.1015625, 0.6552734375, 1.615234375, -0.230224609375, 0.90673828125, -0.049468994140625, -0.4423828125, -1.1376953125, 1.4443359375, -0.505859375, 0.6640625, -0.277587890625, -2.18359375, -0.54345703125, -0.4453125, 0.31982421875, 0.57763671875, -2.22265625, -1.4443359375, -0.2481689453125, -0.087158203125, 0.40283203125, 0.24365234375, 0.1839599609375, -1.1748046875, -0.80126953125, 1.2822265625, 0.60546875, -0.9404296875, -2.69921875, -0.40625, 0.384033203125, 1.3828125, 0.08984375, 1.275390625, -0.6953125, -1.3837890625, -0.09503173828125, 0.08624267578125, -1.1748046875, 0.55322265625, -0.97216796875, 1.1484375, -1.3330078125, 0.63720703125, -1.0791015625, 0.2110595703125, -1.484375, 0.4814453125, 1.6494140625, -0.517578125, 2.310546875, -0.0235595703125, -1.84375, -0.5732421875, -0.3095703125, -1.6005859375, -1.7822265625, 0.0034503936767578125, -0.2353515625, 0.9189453125, -0.42919921875, -0.443115234375, 0.5341796875, -1.1923828125, 1.0439453125, 0.3916015625, -0.235107421875, 0.1729736328125, 1.15234375, 0.475341796875, 0.37646484375, -0.2095947265625, 0.98388671875, -0.240478515625, -0.97265625, 0.1285400390625, 1.205078125, -0.83642578125, 1.0166015625, -0.80078125, 0.1029052734375, 1.6943359375, -0.35205078125, 0.83642578125, -0.056182861328125, 0.08843994140625, 0.611328125, -0.060516357421875, 0.098388671875, 1.8232421875, -1.3662109375, -0.787109375, -0.8427734375, -0.053314208984375, -0.031280517578125, -1.6630859375, 0.93212890625, 0.98779296875, 0.349853515625, -1.7607421875, 0.9052734375, -0.1912841796875, 0.76806640625, -0.1544189453125, 1.2392578125, 1.7060546875, -1.16796875, -1.2685546875, -1.8427734375, -1.744140625, 1.4365234375, 0.355712890625, 0.1337890625, 1.109375, 1.6875, -0.157470703125, -0.5634765625, 0.19580078125, 1.2724609375, 0.332275390625, -0.787109375, -0.63330078125, -0.305419921875, -0.490478515625, -0.716796875, 1.287109375, -0.6728515625, 0.077392578125, 0.67041015625, -1.3857421875, -0.236328125, -0.82568359375, -0.47998046875, 0.1436767578125, -0.77978515625, 1.287109375, 0.1749267578125, 1.09765625, 1.4033203125, -1.4189453125, 0.341552734375, 1.0537109375, -0.285400390625, 1.0634765625, -0.441650390625, 0.389404296875, -0.40673828125, -1.64453125, -0.68115234375, 0.66845703125, 0.5498046875, 0.164794921875, -0.043792724609375, 0.29931640625, 1.005859375, 2.6484375, -0.62646484375, -0.1983642578125, 1.0498046875, -0.68017578125, 0.1409912109375, 0.41455078125, -0.95849609375, 1.0908203125, 0.65087890625, 0.123291015625, -0.386962890625, -1.818359375, -1.033203125, 0.66943359375, 1.5859375, -1.037109375, 0.06829833984375, -2.099609375, -0.18359375, -0.77197265625, 0.873046875, 0.66259765625, 1.1396484375, -0.57861328125, 0.1964111328125, 0.74658203125, 1.1103515625, -0.9345703125, -0.2303466796875, 0.0833740234375, -0.9609375, 0.0478515625, -0.41796875, 1.638671875, -0.268310546875, 0.552734375, 0.69140625, -1.0224609375, -0.52001953125, -1.42578125, 0.98486328125, 0.6162109375, -0.440185546875, 0.1966552734375, 0.2449951171875, 1.9150390625, 0.779296875, 0.52490234375, 1.84375, -1.2900390625, 1.9384765625, 2.07421875, -2.115234375, -0.250732421875, -0.439453125, -0.71240234375, 0.0712890625, -0.335205078125, 0.089599609375, -0.412353515625, -0.08782958984375, 0.89697265625, 1.21875, 1.431640625, -0.732421875, -0.71484375, -0.80712890625, 0.43017578125, 0.39501953125, -1.4716796875, 0.254150390625, 0.97900390625, 0.46630859375, 0.2442626953125, 1.19921875, 0.76513671875, -0.85595703125, 0.8740234375, -0.0176544189453125, 1.935546875, -1.5732421875, -0.466796875, -0.66015625, -0.8232421875, -0.90380859375, 0.1856689453125, -0.81982421875, 0.28271484375, -0.1470947265625, 1.591796875, -0.438232421875, 0.3818359375, 0.7626953125, -1.5244140625, -0.198974609375, 1.3984375, 1.7666015625, -0.7041015625, 0.33642578125, 0.021209716796875, -1.0703125, 1.2265625, -3.46484375, 0.10577392578125, 0.73291015625, -1.353515625, -0.1126708984375, 1.2509765625, 0.6962890625, 0.64501953125, 1.3974609375, -0.329345703125, -2.048828125, -0.63232421875, -0.26318359375, -0.599609375, -0.047210693359375, -0.292236328125, -0.141357421875, -0.93505859375, -1.28515625, -0.1202392578125, -0.282470703125, 1.6201171875, 0.52294921875, 0.607421875, 0.12060546875, 0.294677734375, -0.27001953125, 0.4072265625, 1.2421875, 0.332275390625, 0.66650390625, 1.6689453125, 0.125244140625, -0.373779296875, 2.5234375, -1.15234375, 0.15380859375, 1.61328125, 0.36865234375, -0.327880859375, 0.60498046875, 0.876953125, -2.162109375, 0.057952880859375, 0.319091796875, -2.390625, 0.68603515625, -0.298583984375, -0.86376953125, -0.2177734375, -0.2362060546875, 0.219482421875, -0.00437164306640625, -2.431640625, 0.37939453125, 0.3349609375, -0.7978515625, 1.341796875, -0.020416259765625, 0.67041015625, 0.12200927734375, -0.1632080078125, -1.681640625, -0.20068359375, -0.435791015625, -1.2568359375, 0.5673828125, -1.1728515625, 0.5380859375, -0.8837890625, 0.306640625, -0.2666015625, -3.4453125, 0.346923828125, 0.5361328125, 0.69140625, -1.8583984375, -0.22607421875, 0.117919921875, 0.066650390625, 0.84130859375, 1.939453125, 1.39453125, -1.1552734375, 1.4208984375, 0.91162109375, 0.982421875, -0.861328125, 0.82763671875, -0.7431640625, -0.458251953125, 0.351318359375, 1.9560546875, 0.8212890625, 0.2181396484375, 1.4912109375, 1.4619140625, -0.1920166015625, 1.3974609375, -0.2232666015625, 0.9013671875, -0.7109375, 0.90576171875, 0.35595703125, 1.1826171875, 0.65478515625, -0.7294921875, -0.58349609375, 1.7841796875, 0.10968017578125, 0.9853515625, 0.955078125, 0.250732421875, -0.39599609375, -0.2408447265625, -1.0615234375, 0.26806640625, 0.876953125, 1.4033203125, 0.229248046875, -1.0146484375, 1.8330078125, -2.318359375, 1.4345703125, 0.65283203125, 0.67919921875, 0.98046875, -0.43798828125, 0.48193359375, 0.01523590087890625, 0.09844970703125, 0.23046875, 0.1341552734375, 0.55908203125, -1.7353515625, 0.87158203125, 1.8310546875, 0.970703125, -0.71875, -0.468017578125, -2.0234375, -0.1739501953125, 1.0, 0.9482421875, 0.5625, -0.32666015625, 0.130859375, 0.7734375, -1.3095703125, 1.421875, 0.05133056640625, -1.9521484375, 0.7724609375, -2.396484375, -2.14453125, -1.734375, 0.12890625, 2.125, 1.033203125, 0.170654296875, -1.5087890625, 0.479736328125, -0.5380859375, -0.71484375, -1.341796875, -0.6298828125, -0.87744140625, 1.4599609375, 0.211669921875, 0.07061767578125, 0.6865234375, 0.6611328125, -0.272216796875, -1.251953125, -0.57568359375, -0.025726318359375, -0.1715087890625, 1.326171875, 0.379150390625, -1.0712890625, -0.1510009765625, 0.0828857421875, -0.293212890625, 1.6923828125, -0.07757568359375, -0.390380859375, 0.40869140625, -1.1435546875, 0.814453125, 0.315673828125, 1.326171875, -0.1358642578125, -0.52783203125, -1.7080078125, 0.50244140625, 0.6689453125, -0.9453125, 1.7041015625, -0.351318359375, 1.2333984375, 0.1329345703125, 1.1953125, -2.02734375, -1.56640625, -1.5556640625, 0.09527587890625, 0.310791015625, 2.36328125, 0.5927734375, 0.53125, 0.44140625, -0.29345703125, 0.350830078125, 0.30078125, 0.0487060546875, -0.66748046875, 1.958984375, 1.6748046875, -1.705078125, -0.1744384765625, -0.448486328125, 1.6689453125, 0.197265625, -2.6328125, -0.1839599609375, 2.025390625, -1.2421875, 1.2578125, 2.404296875, 0.286376953125, 0.9052734375, 1.3427734375, 2.1171875, -0.23095703125, 0.237060546875, -1.4072265625, -2.2421875, 0.2197265625, 0.028045654296875, -1.84765625, 1.6943359375, -0.572265625, 0.211181640625, -0.951171875, -0.472900390625, 0.44873046875, 0.46826171875, -0.705078125, -0.239013671875, 0.71435546875, 0.379150390625, -1.03515625, 0.8330078125, 1.005859375, -0.487548828125, 0.3671875, -0.923828125, 0.39599609375, 0.09906005859375, 0.52099609375, 1.1513671875, -0.5478515625, 0.0806884765625, 0.440673828125, -1.3291015625, 1.037109375, 3.171875, 0.5078125, 0.095458984375, -1.138671875, 1.0390625, 0.427490234375, 1.0029296875, 0.32763671875, -1.7431640625, 1.1474609375, -0.626953125, -0.669921875, 0.25244140625, 1.3623046875, -0.36767578125, -0.2587890625, -0.284912109375, -0.5810546875, -0.39697265625, 0.9892578125, -0.8134765625, -0.919921875, 1.1025390625, 0.381591796875, 0.66064453125, -0.36962890625, -0.00518035888671875, -0.375, -0.52685546875, 0.72607421875, 0.364013671875, 0.63720703125, 0.137451171875, -3.3359375, 0.83740234375, -0.62646484375, -0.10675048828125, -0.062255859375, 0.62255859375, -0.994140625, 0.406005859375, -1.5771484375, 1.1669921875, -1.150390625, 0.7880859375, 0.9560546875, 0.77978515625, 1.0673828125, -0.28955078125, -0.705078125, -0.68408203125, -0.79833984375, 1.654296875, -0.403076171875, -0.1192626953125, 0.362060546875, -1.103515625, 1.5224609375, 0.473388671875, -0.7490234375, -0.7314453125, -1.287109375, 0.83740234375, -0.51953125, -0.9619140625, -0.705078125, 0.6181640625, -0.5732421875, -0.35546875, 0.875, -1.5126953125, 1.9423828125, 1.9580078125, -1.3349609375, 0.71484375, -0.1510009765625, 0.476318359375, -0.9443359375, -0.42578125, 0.3251953125, 0.150146484375, 0.0771484375, -1.90625, -0.7119140625, -1.18359375, -1.556640625, 0.11834716796875, 1.1806640625, -0.59228515625, -0.6650390625, 0.75830078125, 0.287109375, 1.17578125, 1.859375, -0.69775390625, 1.3486328125, -0.54345703125, 0.45166015625, 0.81298828125, 0.79833984375, 2.208984375, -1.2607421875, -0.92333984375, -1.9423828125, -0.962890625, 0.66796875, -1.0283203125, 1.611328125, 0.333984375, 0.041473388671875, 0.99462890625, 0.1683349609375, -2.357421875, 0.22607421875, 0.39208984375, -0.1065673828125, -0.78466796875, -0.490478515625, 0.3515625, 0.1788330078125, 1.3876953125, 0.3583984375, -0.1583251953125, 1.19140625, 1.5, 0.6767578125, 1.517578125, -0.91943359375, -1.8837890625, 0.62353515625, 0.370849609375, 0.2392578125, 0.304931640625, 1.5078125, -0.94921875, -0.175537109375, 1.6396484375, -0.88427734375, 0.418701171875, -1.3017578125, 0.89453125, -1.0517578125, 1.693359375, -0.313720703125, 1.17578125, -0.386962890625, 0.36279296875, -1.2890625, -0.12176513671875, 0.34521484375, 0.9091796875, 1.6982421875, 0.552734375, -0.9609375, -0.8916015625, 0.591796875, 0.61669921875, 0.55517578125, 0.892578125, -0.12042236328125, -0.994140625, 0.56689453125, 0.08575439453125, 0.032958984375, -0.23486328125, 0.06549072265625, -0.6728515625, 0.40087890625, 0.96630859375, -0.02838134765625, 0.6484375, -0.059326171875, 0.362060546875, 0.55810546875, 0.5380859375, -0.51220703125, -0.9140625, 1.4345703125, -2.759765625, 0.60546875, 0.411865234375, 0.86865234375, 0.7060546875, -1.4248046875, -2.544921875, -0.24658203125, -0.189453125, -0.57470703125, 0.5693359375, -0.4443359375, 0.23388671875, 0.294189453125, -1.2841796875, 0.318115234375, 1.408203125, -0.420166015625, -0.056915283203125, -1.21484375, -0.473388671875, 0.58203125, 0.288818359375, 0.91650390625, 0.183349609375, -0.55712890625, -1.955078125, -0.486572265625, -0.293212890625, -0.80078125, -0.79248046875, -1.0634765625, 1.1962890625, 0.75244140625, 0.0716552734375, 0.414306640625, 0.168212890625, -0.888671875, -0.77880859375, 0.95654296875, 0.271240234375, -0.91552734375, 0.96923828125, 0.5234375, 0.54345703125, 0.29931640625, -1.322265625, 0.36279296875, -0.7392578125, -0.580078125, 0.357421875, 0.395751953125, -0.386962890625, -0.5771484375, -0.78271484375, -0.61474609375, -0.327880859375, -0.1934814453125, -0.75341796875, -0.01197052001953125, -1.3662109375, -1.921875, -0.178466796875, -1.2978515625, -0.28955078125, 1.703125, -0.54833984375, 0.95751953125, 0.83056640625, -0.54150390625, -0.8466796875, 0.28857421875, 0.159912109375, -0.9921875, 1.794921875, -1.5869140625, 1.2109375, -0.7119140625, -0.82177734375, -0.08270263671875, -0.2122802734375, 0.261962890625, -0.978515625, -1.1298828125, -0.6904296875, 0.286376953125, 0.958984375, -0.50341796875, 0.475341796875, -1.4697265625, -0.88134765625, 0.058197021484375, -0.939453125, 1.375, -0.5146484375, -0.7841796875, -1.126953125, -0.46044921875, -0.74560546875, 0.312255859375, 0.11016845703125, -0.0843505859375, 0.268310546875, 1.384765625, 0.60595703125, 0.1998291015625, -0.11053466796875, 0.640625, 0.74365234375, 0.8505859375, 1.0400390625, 1.5693359375, 0.52978515625, -1.0634765625, -0.35791015625, 2.1796875, 0.57568359375, 0.87939453125, 0.169677734375, -0.134033203125, -0.50537109375, 0.0270538330078125, 0.693359375, -0.83642578125, 0.251708984375, 1.6787109375, 1.6884765625, -0.6552734375, 0.7255859375, 0.57861328125, 0.2481689453125, -0.12158203125, -0.09613037109375, 2.11328125, -0.278564453125, 0.7275390625, -0.1160888671875, 0.0202178955078125, 0.29296875, -0.135986328125, -0.225341796875, -0.7109375, -1.4501953125, -0.85791015625, 0.239990234375, -0.1636962890625, 2.138671875, 0.908203125, -0.3408203125, 0.87060546875, 1.6142578125, 0.9228515625, -0.52734375, -0.1258544921875, 0.837890625, -0.296875, -0.341552734375, -0.79052734375, 0.2147216796875, 1.4765625, -0.420654296875, 1.7197265625, 0.85107421875, 0.333984375, -1.1025390625, -0.470947265625, -0.44091796875, -2.189453125, 1.3828125, 0.62109375, -0.60888671875, -0.61474609375, -0.82275390625, 0.255859375, 0.5615234375, 0.720703125, 1.04296875, -0.546875, 0.200439453125, 0.359375, 0.76806640625, -0.278564453125, -1.3056640625, 0.73681640625, -0.1588134765625, 0.90478515625, -0.2318115234375, 0.634765625, 2.322265625, -0.4130859375, -1.0361328125, 0.2176513671875, 0.48779296875, 0.135009765625, -1.529296875, -1.2841796875, 0.312255859375, -1.3447265625, -0.261962890625, -0.10516357421875, -0.5380859375, -0.8515625, -1.1474609375, -1.1904296875, -0.377197265625, -1.4501953125, -0.8369140625, -1.3603515625, 0.1292724609375, -0.216064453125, 1.8837890625, -0.1590576171875, -2.01953125, -1.0390625, -0.380859375, 0.34375, 1.861328125, 2.08203125, 0.3837890625, 0.9716796875, 0.472900390625, 1.44921875, 0.80712890625, -0.89599609375, -0.705078125, -1.3876953125, 0.1502685546875, 0.1622314453125, -1.189453125, 0.316650390625, 0.693359375, 0.7880859375, -1.1015625, 0.97509765625, -0.6787109375, -1.1298828125, -0.32666015625, 0.483642578125, 1.0908203125, 0.9765625, 0.1795654296875, 0.57568359375, 2.046875, 1.052734375, 0.330810546875, -0.43359375, -0.3671875, -0.370361328125, 0.0684814453125, 0.51806640625, -1.1572265625, 0.826171875, 0.82470703125, 1.7958984375, -0.099365234375, 0.40478515625, -0.307861328125, 0.300048828125, -1.462890625, -0.35205078125, -0.28955078125, 0.1026611328125, 0.11004638671875, 0.4443359375, 0.5556640625, -0.36376953125, 0.66943359375, -2.060546875, 0.57763671875, 0.71044921875, 0.5087890625, -1.4521484375, -0.947265625, -0.90478515625, 0.0677490234375, -1.296875, -0.87890625, -0.08746337890625, -0.09100341796875, 0.6640625, 1.61328125, 1.61328125, 0.57861328125, -0.79736328125, 0.6982421875, 0.2091064453125, -0.28125, 0.1710205078125, 0.437255859375, -0.74267578125, -0.53125, 0.1956787109375, 0.336669921875, 0.2080078125, -0.248779296875, 1.556640625, 1.3359375, 1.5673828125, -0.352294921875, -0.2125244140625, -1.2490234375, -0.91943359375, -0.0323486328125, 0.828125, 0.157958984375, 0.06201171875, -1.068359375, 0.34033203125, -1.08203125, 0.6845703125, 0.0633544921875, -1.6875, -0.44873046875, -0.90478515625, -0.29541015625, 1.6005859375, -0.81396484375, -0.978515625, 1.5537109375, 0.93896484375, 1.015625, -0.236328125, 0.4208984375, -0.5107421875, 1.3642578125, -2.375, -0.306884765625, -1.2080078125, 1.4072265625, -2.06640625, -2.009765625, 1.333984375, 0.1319580078125, -0.43896484375, -0.8740234375, -1.375, -0.416259765625, 0.6533203125, 0.2493896484375, 1.271484375, -0.1517333984375, 0.85986328125, 0.78515625, -0.9580078125, -0.3583984375, 0.60302734375, -0.978515625, -0.72802734375, 0.29443359375, -0.038818359375, -0.69873046875, -0.865234375, 1.326171875, -0.54150390625, -0.0267791748046875, 1.2333984375, 0.92822265625, 0.81005859375, -0.299560546875, 0.75927734375, -0.67431640625, -1.4580078125, 0.467529296875, 0.578125, 0.239013671875, 0.654296875, 1.01953125, 0.5712890625, 2.228515625, -1.0166015625, -0.37255859375, -0.50048828125, 0.451416015625, 0.39599609375, 1.515625, 0.2044677734375, 0.61572265625, 0.05224609375, 0.74169921875, -0.62158203125, 1.755859375, 0.82763671875, -0.350830078125, 1.1455078125, -0.492431640625, -0.302978515625, 0.495849609375, -0.41259765625, -0.51953125, -0.6826171875, 1.0400390625, 1.298828125, -0.1544189453125, -0.01100921630859375, 0.493408203125, 1.283203125, 0.5966796875, -0.047821044921875, -0.61865234375, 0.177978515625, 0.94970703125, -1.1748046875, -0.253173828125, -0.6708984375, -0.282470703125, -0.249755859375, -0.80810546875, -0.188720703125, -1.1650390625, 0.79931640625, -2.31640625, 0.263427734375, -0.49951171875, -1.556640625, 1.533203125, 1.05859375, -0.1812744140625, 0.4658203125, 2.037109375, 1.1689453125, 1.3232421875, -1.4541015625, -0.89208984375, -0.689453125, 2.56640625, -1.40234375, 2.0859375, 0.0865478515625, -0.51220703125, -0.21142578125, -1.02734375, 1.080078125, -0.2039794921875, -0.92529296875, 0.12396240234375, -0.324951171875, -1.7431640625, -0.68505859375, -0.36767578125, -0.70751953125, 0.89599609375, -0.671875, -0.83349609375, -1.044921875, -0.65478515625, 0.0244903564453125, 1.3974609375, -0.716796875, -1.474609375, -0.57861328125, 2.107421875, -0.52197265625, -0.200439453125, -0.142578125, -0.736328125, -0.31591796875, 1.365234375, -0.2186279296875, -0.74072265625, -0.10784912109375, 0.1524658203125, -0.438232421875, 0.888671875, -0.58642578125, -1.4384765625, -1.158203125, 0.9541015625, 1.0693359375, 0.884765625, -0.0919189453125, -2.111328125, 0.041839599609375, 1.1494140625, 0.7705078125, -0.2171630859375, -0.485107421875, -1.9619140625, -0.52392578125, -1.46875, -1.080078125, 2.244140625, -0.44873046875, -0.473388671875, 0.38623046875, -0.11309814453125, 0.041107177734375, 1.0361328125, -1.0947265625, -1.7978515625, 0.4267578125, -0.51904296875, 0.484130859375, 0.919921875, -0.251220703125, 1.7578125, 0.39697265625, 1.0498046875, 1.072265625, -0.50927734375, 0.88671875, -0.1121826171875, -1.185546875, -0.5400390625, -0.1715087890625, -1.033203125, 2.599609375, 0.6455078125, -0.71630859375, 0.385498046875, -0.414794921875, 1.4326171875, 1.1396484375, -0.388671875, 0.160400390625, 0.36181640625, 0.974609375, -0.147216796875, 1.564453125, -0.0213165283203125, -0.04730224609375, 0.70458984375, -0.2310791015625, -1.3408203125, -0.189697265625, -0.33056640625, -0.2012939453125, 0.58837890625, 0.83935546875, -0.08123779296875, 0.2298583984375, -2.056640625, -1.3544921875, -0.87841796875, -1.447265625, -0.449462890625, 0.85009765625, 0.135009765625, 1.7216796875, 0.481689453125, 0.10675048828125, -0.96533203125, -0.356689453125, -0.1373291015625, 0.857421875, 0.10980224609375, 0.697265625, -0.2396240234375, -0.51513671875, -1.015625, 1.12109375, -0.662109375, 0.01540374755859375, -0.1824951171875, -1.3701171875, -0.10870361328125, 0.111083984375, 0.568359375, -0.98291015625, 1.9091796875, 0.2393798828125, 0.87890625, 0.240234375, 0.65087890625, 0.80859375, -1.0537109375, 1.287109375, 0.9912109375, -0.64404296875, 0.82275390625, 0.0667724609375, -0.017547607421875, 0.83984375, 0.04022216796875, -3.84765625, 0.0227508544921875, 0.76611328125, -0.68896484375, -0.378173828125, 1.9462890625, -0.51025390625, 0.492431640625, 0.97509765625, -0.2452392578125, 1.8544921875, -0.081298828125, 1.466796875, -2.625, -0.74658203125, 1.3251953125, 0.27392578125, -1.009765625, 0.2220458984375, -0.321044921875, 0.88720703125, 0.29150390625, 1.0791015625, -1.0, 0.8662109375, -1.189453125, 0.08734130859375, -0.84375, -1.1025390625, 0.74951171875, 1.0693359375, 2.431640625, -0.9375, 0.09100341796875, 0.55615234375, -0.4326171875, -0.7998046875, 0.72900390625, 0.06585693359375, 0.90625, -1.6728515625, 0.363037109375, 0.7490234375, 0.215087890625, -0.79150390625, -0.74365234375, 1.35546875, -0.619140625, -0.47900390625, 0.22705078125, 0.814453125, 0.76171875, -0.49560546875, 1.587890625, 0.331298828125, 0.5771484375, 0.3525390625, -0.197509765625, 0.2109375, 0.865234375, -0.7470703125, -0.74609375, 0.8291015625, -0.55908203125, 3.376953125, 0.452880859375, 1.3759765625, 1.958984375, 1.2998046875, 0.7119140625, -0.70849609375, -0.810546875, 0.25341796875, 0.54541015625, -2.673828125, 0.63818359375, 1.0595703125, -2.12109375, -0.533203125, -0.9189453125, 0.12646484375, 0.8408203125, 0.26025390625, 0.587890625, -1.5625, -0.48486328125, 1.34765625, -1.0556640625, -0.78662109375, -0.1058349609375, 0.8759765625, -1.3779296875, 0.9462890625, 0.67041015625, -0.360107421875, 0.2734375, -0.89208984375, -0.6708984375, 0.7021484375, 0.67822265625, -0.79150390625, 0.03656005859375, 1.2705078125, 0.765625, -1.3349609375, 0.5234375, -1.2333984375, 0.3466796875, 0.43798828125, 1.341796875, 0.028717041015625, 0.580078125, -0.73779296875, -0.994140625, 1.8037109375, 0.4482421875, -0.6650390625, 0.2261962890625, 0.6669921875, -1.0615234375, 0.81640625, 0.330322265625, -1.4931640625, -0.347412109375, -0.5263671875, 0.92578125, 1.3115234375, 1.5634765625, -0.7216796875, 0.544921875, 1.2568359375, 0.16845703125, 0.75390625, -0.6875, 0.21533203125, 2.4140625, 0.302001953125, 1.1865234375, 1.1083984375, -0.295654296875, 0.1553955078125, -1.90234375, -0.81103515625, 1.7080078125, -0.6376953125, -2.060546875, 0.23388671875, 0.1561279296875, 1.314453125, 0.04791259765625, 0.318115234375, -0.048095703125, 0.71875, -1.4296875, 1.4501953125, -1.1982421875, -1.0146484375, -0.136962890625, 1.9169921875, -2.3046875, -0.1932373046875, -0.92626953125, -1.53125, 0.329345703125, -0.93603515625, 0.1268310546875, 1.15234375, 0.179443359375, 0.4990234375, 2.501953125, -0.83984375, -0.33447265625, -0.173095703125, -0.001316070556640625, 0.2900390625, -1.2919921875, -2.18359375, -1.3466796875, -0.81201171875, 1.27734375, 1.4091796875, -0.373291015625, -1.0439453125, -0.85546875, -0.07501220703125, 0.32177734375, 1.361328125, 0.1922607421875, -1.109375, -0.1605224609375, -0.85302734375, 0.50537109375, -1.1357421875, -1.162109375, 1.03125, -0.2783203125, 2.12890625, 1.125, 1.3203125, 0.1510009765625, 0.12744140625, -0.2459716796875, 0.9638671875, 0.1336669921875, -0.5, -0.94775390625, 0.0821533203125, 0.7373046875, -1.2158203125, 1.33984375, -0.74658203125, 0.479248046875, 1.052734375, -1.791015625, 0.74267578125, 0.158203125, 1.0078125, 0.56884765625, 0.1400146484375, 0.935546875, -0.98583984375, -0.2481689453125, 0.6552734375, 0.08538818359375, 1.0517578125, -0.128662109375, -0.1170654296875, -0.353515625, 0.7548828125, -0.4423828125, 0.54345703125, -0.65185546875, -0.39013671875, 0.52587890625, 1.12890625, -0.8115234375, -0.3857421875, 0.36181640625, -0.580078125, -0.0040435791015625, 0.73681640625, 0.61669921875, -0.024932861328125, 0.77734375, 0.13232421875, -0.406982421875, -0.18798828125, 0.479736328125, -0.42041015625, 0.09600830078125, -0.60009765625, 1.458984375, 1.96484375, -0.269287109375, 0.2398681640625, 0.95654296875, 1.1142578125, 0.316650390625, -1.4541015625, -1.15625, -0.7568359375, 0.472412109375, 0.92138671875, 0.93310546875, -1.4306640625, -1.44921875, -1.234375, -0.0980224609375, -1.98046875, -0.0823974609375, 0.33056640625, 0.62939453125, -0.44921875, -1.8837890625, -1.6318359375, -1.224609375, -0.061737060546875, -0.8486328125, 1.220703125, -1.3701171875, 1.0478515625, 0.1265869140625, 1.7060546875, -0.59375, -0.94775390625, 0.358154296875, 0.5263671875, 0.058807373046875, 1.29296875, -0.15576171875, 1.0322265625, 0.84228515625, 0.8173828125, -0.779296875, -1.0322265625, 0.386474609375, 0.51513671875, 0.9326171875, 0.017425537109375, 0.66259765625, 0.94189453125, 1.2001953125, -0.9453125, 1.39453125, 0.03857421875, 2.251953125, -0.5205078125, -1.9189453125, 0.572265625, -1.2353515625, 0.1588134765625, 2.037109375, -1.75, -0.0888671875, -0.6171875, 1.189453125, -0.9892578125, 1.1708984375, -0.1458740234375, 1.376953125, 1.322265625, 0.94287109375, 0.253662109375, -1.953125, -1.130859375, 1.1181640625, 0.0307464599609375, -0.74072265625, 0.04583740234375, -0.1556396484375, 0.378173828125, -1.1083984375, -0.38525390625, -0.417236328125, -1.357421875, 0.7978515625, -2.4375, 1.802734375, 1.6494140625, -0.7900390625, 1.9443359375, -0.225341796875, 0.4677734375, -1.939453125, 0.494873046875, 0.328125, -0.43896484375, -0.430419921875, 0.69873046875, 0.54296875, -0.5771484375, 0.81982421875, 1.14453125, 1.55859375, -0.051422119140625, 0.83642578125, 1.6943359375, 0.69921875, -0.10345458984375, 0.2186279296875, 1.8583984375, 0.269775390625, -0.21630859375, 1.6689453125, -0.04937744140625, 0.74072265625, -0.8671875, -0.6533203125, -0.1876220703125, -0.98681640625, 0.68505859375, -0.1727294921875, 0.45556640625, 0.01000213623046875, -0.1650390625, 1.84375, 0.452880859375, 1.4091796875, -0.93896484375, 1.7578125, 0.7646484375, -0.411376953125, 0.380615234375, 2.826171875, -0.316650390625, -0.63427734375, 1.068359375, -0.7060546875, 1.3349609375, -0.55859375, 0.428466796875, -0.040313720703125, -0.6455078125, 0.177490234375, 0.77783203125, 1.015625, -1.087890625, -1.9248046875, 0.1953125, -0.51318359375, -0.144775390625, 0.7255859375, 0.91162109375, -0.0217132568359375, -0.082763671875, -0.39794921875, -0.66748046875, 0.7685546875, 1.5400390625, -0.9501953125, 0.366455078125, 0.2001953125, 1.6416015625, -2.7734375, 1.775390625, -1.5361328125, -0.8271484375, -0.97998046875, -0.35107421875, 0.9501953125, 0.30029296875, -0.30322265625, -2.134765625, -2.400390625, 0.72021484375, 0.474365234375, -1.2998046875, -0.07958984375, 0.385498046875, -0.9541015625, 0.88330078125, 0.1241455078125, 0.68115234375, -0.8828125, -0.58740234375, -0.87744140625, 1.10546875, -0.8095703125, -0.2257080078125, -0.482177734375, -1.162109375, -1.306640625, -0.430908203125, 1.3681640625, 0.2939453125, -0.82568359375, -0.09417724609375, 1.1455078125, -0.46142578125, 0.58349609375, 1.05859375, 0.60107421875, -0.6826171875, -1.11328125, 0.34375, 1.0068359375, 1.5390625, 1.3251953125, -0.1702880859375, 0.97119140625, -0.546875, 1.3720703125, -2.189453125, 0.78662109375, 0.427490234375, 0.61865234375, -0.7353515625, -0.4365234375, 0.6552734375, -0.64453125, -0.140625, 0.251220703125, 0.10540771484375, 0.310302734375, 0.04742431640625, -0.71142578125, 0.89208984375, 0.368408203125, -0.1710205078125, -0.66796875, -1.3076171875, -0.1207275390625, -0.8046875, -0.78125, 0.46240234375, 0.40185546875, -0.44873046875, -0.444580078125, -1.0869140625, -0.79638671875, -1.3447265625, -0.096435546875, 0.654296875, -2.8984375, -1.0703125, 0.396728515625, -0.96875, 0.30712890625, 1.013671875, 1.1455078125, 0.286865234375, 0.365478515625, -1.384765625, 0.434814453125, -0.904296875, 0.640625, 0.247802734375, 0.29296875, 0.1995849609375, 1.1552734375, -1.01171875, -0.521484375, -1.6591796875, -0.34619140625, -0.826171875, -0.49658203125, 0.1524658203125, 0.451416015625, 1.0986328125, -0.6005859375, -0.378173828125, -1.720703125, 0.4990234375, 0.552734375, 0.73046875, 0.86474609375, 0.1162109375, 0.108154296875, 0.60107421875, -1.4951171875, 0.37841796875, 1.47265625, 0.6357421875, -0.197509765625, -0.92529296875, -0.12548828125, -0.13134765625, 0.8291015625, -1.478515625, -0.7509765625, 3.078125, -1.3583984375, -1.8525390625, 0.4072265625, -3.365234375, 0.123046875, 1.3876953125, -0.07470703125, -1.4736328125, 1.2333984375, 0.2445068359375, -0.83349609375, -0.1474609375, -1.81640625, 1.9599609375, -0.2198486328125, -0.2958984375, 2.57421875, 0.5888671875, -0.93310546875, -0.92822265625, 1.23828125, 0.0228729248046875, -0.59375, 1.1591796875, -0.2071533203125, -0.196044921875, 0.52001953125, -1.1474609375, 2.345703125, -0.36865234375, -0.70751953125, 0.022003173828125, -1.30078125, 0.60302734375, 0.1959228515625, 0.666015625, 1.2578125, 1.0830078125, 1.19140625, 0.53564453125, -0.44189453125, 0.484130859375, 0.0194091796875, -0.6953125, 0.501953125, -0.763671875, 0.87060546875, 1.8759765625, 1.4560546875, -0.1290283203125, 1.7060546875, 1.0166015625, -0.379638671875, -0.77001953125, 1.498046875, 0.75830078125, 0.09051513671875, 0.0784912109375, 1.40234375, 0.63818359375, 0.21630859375, 0.744140625, -1.4892578125, -0.1785888671875, -0.0794677734375, 0.7705078125, -0.77099609375, -0.5849609375, -0.3447265625, -0.63916015625, -0.142578125, -0.5615234375, -0.29345703125, -0.95703125, 0.00890350341796875, 0.57373046875, -0.2337646484375, -1.6201171875, 0.39306640625, 0.1368408203125, -0.95947265625, -1.8115234375, -0.41796875, -0.609375, -1.03125, 0.248779296875, -1.0849609375, 0.357421875, 0.343994140625, -2.220703125, -1.3662109375, 1.2197265625, -0.282958984375, 0.28515625, -0.107177734375, 1.3232421875, -0.408447265625, 0.2254638671875, 1.1416015625, 0.404052734375, 0.39404296875, -0.09124755859375, 0.77783203125, -1.40625, 1.1005859375, -0.05181884765625, -0.29833984375, 0.81640625, 0.179443359375, -0.0908203125, -0.88232421875, 0.88330078125, 0.509765625, -0.07769775390625, -0.85400390625, 0.1307373046875, 1.234375, -0.268310546875, -0.9208984375, 0.6494140625, 0.94140625, -1.1748046875, 0.982421875, 0.2369384765625, 1.15234375, 0.0518798828125, 0.267822265625, -1.1904296875, 0.66943359375, -0.5380859375, 0.0869140625, -0.74609375, 1.16796875, 0.57275390625, -1.0390625, -0.406982421875, 1.46875, -0.479248046875, -1.3583984375, -2.07421875, -1.119140625, 0.720703125, 1.0703125, -1.12890625, 0.364501953125, 0.294921875, -1.3681640625, 0.923828125, -1.0859375, -1.3125, -0.048583984375, -1.3984375, 1.2724609375, -0.70263671875, 0.0733642578125, 1.146484375, -0.00592041015625, -0.490478515625, 0.5810546875, 0.422607421875, -0.2213134765625, 0.56005859375, 0.4873046875, -0.58837890625, -0.59375, 0.51416015625, 0.328125, -0.12298583984375, -0.247802734375, 0.4345703125, -0.7275390625, 1.279296875, -1.86328125, 2.181640625, -0.69140625, -0.4501953125, 0.07080078125, -0.434326171875, -1.611328125, 2.513671875, -1.5302734375, 0.2939453125, -0.209228515625, -0.58935546875, -0.7265625, -0.07354736328125, 1.6337890625, -0.5205078125, -0.01155853271484375, 0.80859375, -0.1162109375, 0.470458984375, 0.9189453125, -0.47314453125, 0.94140625, 0.155029296875, -0.89794921875, 0.93408203125, -0.80810546875, -1.5810546875, 1.86328125, 0.59033203125, -0.9267578125, -0.50341796875, -0.2442626953125, 1.283203125, 0.60107421875, -0.66162109375, 1.5947265625, 0.05902099609375, -0.095947265625, 0.02325439453125, 0.78955078125, 0.25146484375, -1.6376953125, 0.35498046875, -3.03125, -2.09765625, -0.71923828125, -0.1585693359375, -0.490234375, 1.9638671875, 0.5166015625, -0.265380859375, -0.199462890625, 0.1688232421875, 2.34765625, 1.8095703125, -0.2008056640625, 1.2431640625, 1.8837890625, 0.185302734375, -0.583984375, -2.578125, 1.337890625, -0.6611328125, 0.01666259765625, 1.9482421875, -1.2353515625, 0.1357421875, 1.1904296875, -1.0751953125, 0.452880859375, -0.94140625, 0.376220703125, -1.6220703125, -1.6611328125, -0.73046875, -1.1025390625, -1.2783203125, -0.00780487060546875, -0.84326171875, 1.0087890625, 1.220703125, 0.256591796875, 1.86328125, 0.736328125, 0.69970703125, 1.0107421875, 0.0137786865234375, -2.333984375, 0.1278076171875, 0.051422119140625, 0.52685546875, 0.56494140625, 0.07440185546875, -0.783203125, -0.10455322265625, 0.05364990234375, 1.7744140625, 0.58837890625, 0.2327880859375, 1.0615234375, -0.8564453125, -0.357666015625, -0.79443359375, 1.185546875, -2.0625, -0.1475830078125, 0.80029296875, 0.2418212890625, -0.35546875, -1.345703125, 1.2939453125, -0.198486328125, -1.9326171875, -0.79296875, 1.9560546875, -0.64208984375, 0.0268096923828125, -1.71875, -0.1749267578125, -0.43115234375, 0.0413818359375, -1.4306640625, -1.7119140625, -1.841796875, -0.056365966796875, -0.853515625, -0.2049560546875, 0.54541015625, -1.576171875, 1.28515625, -1.1142578125, -0.86669921875, 0.5927734375, -0.99072265625, 2.33203125, -0.9638671875, -0.0765380859375, -0.02154541015625, -1.98046875, 1.2802734375, -0.09881591796875, -0.85302734375, 0.68212890625, -2.3515625, -2.716796875, -0.10296630859375, -0.8740234375, 1.51953125, -1.2861328125, 0.97705078125, 0.2352294921875, 1.443359375, 1.8486328125, -0.267578125, -1.3857421875, -1.560546875, -0.0277099609375, -1.5380859375, -0.0906982421875, 0.5439453125, 1.1455078125, -1.10546875, 0.96337890625, 0.393310546875, -0.2362060546875, -0.79443359375, 0.82275390625, -0.054046630859375, -1.6982421875, -0.72705078125, 0.8974609375, -1.046875, 0.1448974609375, 0.3740234375, 1.1298828125, -0.05853271484375, -0.96435546875, -3.05859375, -0.822265625, -0.114990234375, 0.06475830078125, -0.7451171875, 0.458251953125, -1.6220703125, -0.476806640625, -0.497314453125, 0.0660400390625, 0.174072265625, -1.45703125, -0.364501953125, -0.0280914306640625, 1.02734375, 1.009765625, -0.5849609375, 0.1949462890625, 1.1201171875, 0.440185546875, 0.86572265625, 0.400390625, 0.64794921875, -0.43408203125, -0.31689453125, -1.8515625, 0.81005859375, -1.3564453125, 1.140625, 0.237548828125, -0.53564453125, -1.0341796875, 0.66943359375, 0.2489013671875, -1.02734375, -1.06640625, 1.642578125, 0.923828125, 1.2509765625, -0.953125, -0.110595703125, -1.7431640625, -1.08984375, 0.7177734375, 1.72265625, 0.273681640625, -0.646484375, -1.08203125, 1.2255859375, -1.4013671875, -0.43701171875, 1.8349609375, 0.7587890625, 1.697265625, -0.81103515625, -1.4140625, 0.444580078125, -0.26171875, 0.45849609375, -1.1494140625, 0.9609375, 0.60595703125, -1.6787109375, 0.57177734375, 0.6962890625, -0.337890625, -0.7421875, -0.058990478515625, -1.30859375, 0.06402587890625, -0.78662109375, 0.392822265625, -0.194580078125, -1.904296875, -1.9404296875, -0.568359375, -0.1500244140625, -0.79736328125, 0.533203125, 1.8583984375, 0.97705078125, -0.7216796875, 1.5625, 0.3955078125, -1.20703125, 0.62939453125, 0.23779296875, -0.84130859375, 0.08245849609375, -1.7900390625, 0.0653076171875, -0.9462890625, 0.70751953125, 1.0986328125, 0.3974609375, 0.6171875, -1.5546875, -0.88671875, 1.0546875, -2.103515625, 1.2744140625, -1.5537109375, -0.67578125, 0.222900390625, -0.453369140625, -0.119140625, 0.4169921875, 0.1417236328125, 1.41796875, -0.796875, 1.4501953125, -0.8759765625, 0.295166015625, 0.2471923828125, 2.15234375, 1.28125, -1.16796875, 0.15234375, -1.1005859375, -2.123046875, -1.158203125, -2.28515625, 0.96337890625, -1.6220703125, 0.465087890625, 0.55419921875, 1.2568359375, 0.349853515625, 0.56396484375, -0.640625, -1.1474609375, 0.47314453125, -0.99609375, -0.80224609375, -1.44140625, -2.1640625, -0.46240234375, -0.78515625, 0.849609375, -0.404296875, -0.229736328125, -0.65380859375, 0.25927734375, 1.3603515625, -1.4765625, -0.5537109375, -1.7373046875, 0.35693359375, 0.7744140625, 2.865234375, -1.037109375, -0.8720703125, -0.7021484375, -1.2802734375, 0.796875, 1.412109375, -0.2763671875, 0.6201171875, 0.8623046875, 0.160888671875, -0.0802001953125, 2.013671875, 1.216796875, 0.7080078125, 0.309814453125, 0.297607421875, 0.7099609375, 0.1627197265625, 0.5888671875, -1.0361328125, 0.284912109375, -0.7685546875, 0.76123046875, 0.87353515625, 0.779296875, -0.49609375, 1.78125, 1.6552734375, -0.499267578125, 0.7568359375, -0.36669921875, -0.26611328125, -0.86962890625, -1.8388671875, 0.52197265625, 1.3525390625, -2.798828125, -0.49072265625, 1.0478515625, -0.66162109375, 0.470703125, -1.681640625, -0.66259765625, -0.281982421875, 0.70263671875, -1.3984375, -1.5810546875, -1.88671875, 1.263671875, -1.61328125, -0.37060546875, -0.642578125, -0.4033203125, 0.787109375, -0.367431640625, 1.283203125, -0.65087890625, 0.036163330078125, 0.61181640625, 0.32763671875, 0.31396484375, 0.955078125, -0.1512451171875, -0.67236328125, -0.90478515625, -0.348876953125, 1.9921875, 0.071044921875, 0.6142578125, -1.455078125, -0.1524658203125, 1.8095703125, 0.314208984375, -1.7822265625, 0.87060546875, 0.279052734375, -1.1044921875, -0.6728515625, 0.401611328125, -2.029296875, 0.84716796875, -1.4169921875, -0.7919921875, -1.271484375, 0.46728515625, -1.40625, 0.87451171875, 1.0908203125, -1.4990234375, 0.5908203125, 0.47314453125, -0.5556640625, -0.09576416015625, -1.5263671875, 0.262451171875, -0.2939453125, 0.1085205078125, 1.7783203125, -1.8447265625, -1.3740234375, 1.4482421875, 1.212890625, -1.376953125, -1.9169921875, -0.810546875, -0.7998046875, 0.2457275390625, 0.57373046875, -0.343505859375, -0.5048828125, 2.0859375, 0.33154296875, -0.257080078125, 0.05413818359375, 0.353515625, -0.083740234375, 2.12109375, 0.61865234375, 1.3916015625, 0.79443359375, -1.3125, 1.1533203125, -0.7587890625, 0.72314453125, 0.94091796875, 0.1629638671875, 0.61572265625, 0.272216796875, 0.483642578125, 0.78076171875, 0.41162109375, 0.95068359375, -0.859375, 0.7734375, -0.438232421875, -0.2919921875, 0.4619140625, 0.40673828125, -1.46875, 0.69970703125, 1.107421875, 1.466796875, 0.1607666015625, -0.2008056640625, -2.048828125, 0.328857421875, -1.1494140625, -0.340087890625, -1.673828125, 1.07421875, 1.3232421875, -0.50732421875, 0.17333984375, 0.24951171875, -0.06854248046875, -2.15234375, -0.66845703125, 0.68115234375, -0.79443359375, -0.08087158203125, 0.52490234375, 0.857421875, -0.3212890625, 0.60009765625, -1.2294921875, -0.006977081298828125, 0.83056640625, 0.94189453125, -1.9951171875, 0.556640625, -1.6923828125, -0.1822509765625, 0.453125, 1.4501953125, -0.7890625, -0.748046875, -0.2412109375, -0.365234375, 0.466064453125, -0.9033203125, 1.9951171875, 0.45947265625, 0.497314453125, -1.80078125, 0.07275390625, -1.8876953125, -1.2294921875, 0.39697265625, -1.087890625, -0.8076171875, 0.369873046875, 1.71484375, 0.02117919921875, 0.10870361328125, -0.556640625, 1.630859375, 1.619140625, 0.1585693359375, -1.099609375, -0.87158203125, -0.041351318359375, -1.3603515625, 0.82080078125, -0.76318359375, 0.583984375, 0.134521484375, -0.412353515625, 0.42919921875, 0.1920166015625, -0.59716796875, 0.541015625, 0.5341796875, -1.91796875, -0.78369140625, -1.07421875, -0.4111328125, -0.11083984375, -0.1900634765625, -0.0780029296875, 0.8427734375, -1.7314453125, 0.912109375, -0.291259765625, -1.26953125, -0.1822509765625, -1.4306640625, 1.404296875, -0.6748046875, -1.455078125, -0.049530029296875, -2.865234375, 0.08807373046875, 1.1884765625, -0.89501953125, 0.6455078125, -0.3779296875, 2.68359375, 1.4482421875, 0.26806640625, -0.90625, -0.6650390625, 1.5107421875, 0.2171630859375, 1.572265625, 0.9228515625, -0.5234375, -1.525390625, 0.64111328125, 0.411865234375, 0.118408203125, -0.59912109375, -0.0498046875, 0.78955078125, 0.85400390625, -0.1341552734375, -0.04779052734375, -0.247314453125, -0.005748748779296875, 0.1094970703125, -2.607421875, 0.43408203125, 0.022796630859375, -0.92626953125, -0.8076171875, 0.6962890625, 1.6162109375, 0.274658203125, -1.5009765625, 0.837890625, -1.80859375, 1.357421875, -0.58935546875, 0.478271484375, 1.13671875, 1.7236328125, -0.91748046875, -1.2509765625, 0.33203125, 1.189453125, -0.046539306640625, 0.673828125, 2.15625, -2.310546875, -1.0771484375, -1.0361328125, 0.9677734375, 0.78466796875, 0.7119140625, -0.66796875, 0.94580078125, 1.3896484375, -1.0361328125, 0.2236328125, 1.740234375, -0.115234375, -0.287841796875, -0.470703125, -0.55029296875, -1.4521484375, -2.24609375, 0.33740234375, 0.263671875, -0.277587890625, -0.88916015625, -0.485107421875, -0.7119140625, 0.46435546875, 0.022186279296875, 0.09515380859375, -1.677734375, 0.96923828125, -0.8857421875, 1.4482421875, 0.7919921875, 1.1982421875, 0.1021728515625, 1.681640625, -0.71337890625, -0.66015625, 0.49169921875, 1.1064453125, -0.55810546875, -0.1766357421875, -0.92529296875, -0.34619140625, -0.9853515625, -2.41796875, 0.578125, -0.257568359375, 0.58251953125, -0.251953125, -0.427978515625, 1.36328125, -0.564453125, 1.2900390625, -0.556640625, -0.06365966796875, 1.3173828125, 0.036773681640625, -0.2431640625, 1.296875, 1.927734375, -1.0, 2.01171875, -1.6650390625, 0.53125, 0.3662109375, 2.5234375, 0.7939453125, 1.216796875, -1.296875, 2.095703125, 0.55224609375, -0.78173828125, -0.253173828125, 0.2685546875, -0.34814453125, -0.38427734375, -1.4697265625, -0.6982421875, -0.8896484375, -0.1922607421875, -0.1229248046875, -0.0689697265625, 0.27294921875, -0.417724609375, -0.340087890625, -0.0117034912109375, 0.798828125, 0.05810546875, -0.05670166015625, -0.103515625, 0.0504150390625, -0.3037109375, -1.71484375, 0.453369140625, -0.548828125, -0.1309814453125, 0.5966796875, -0.427734375, -0.251953125, -0.04498291015625, -0.5107421875, 0.006801605224609375, -0.65966796875, -1.884765625, -1.9677734375, -0.93310546875, -0.2257080078125, -0.6484375, 0.9853515625, 0.62255859375, -0.41357421875, 0.67919921875, 0.29541015625, -0.1968994140625, 2.09765625, 0.720703125, -0.7919921875, -1.4111328125, -0.17822265625, -2.22265625, -0.71240234375, 1.38671875, 0.07440185546875, -0.441162109375, -0.03839111328125, -1.595703125, 0.23291015625, -0.7412109375, -0.057586669921875, -0.2137451171875, -0.810546875, -2.33984375, 0.89697265625, -0.206787109375, -0.052978515625, 0.060791015625, 0.97021484375, -0.07781982421875, -0.7119140625, 0.6796875, -1.7060546875, -0.92626953125, 0.1417236328125, 0.88623046875, -0.36962890625, -0.68408203125, -1.34765625, -1.3701171875, 0.3984375, 0.0989990234375, -0.47509765625, -0.1142578125, -0.271240234375, -1.08203125, -1.2744140625, -0.127685546875, -0.74267578125, -0.71630859375, 0.050750732421875, -0.395751953125, -0.282470703125, 0.146728515625, 0.495849609375, -0.8056640625, -1.6005859375, -1.2939453125, -0.11669921875, -1.2158203125, 0.033935546875, 0.57568359375, 0.51416015625, 1.1494140625, 0.330322265625, 1.2412109375, 1.228515625, 1.94140625, 0.84912109375, 0.366455078125, -0.439208984375, 1.3955078125, -0.3046875, 0.196044921875, -0.35302734375, 1.03515625, 1.2978515625, -0.477294921875, 0.430908203125, -1.216796875, 0.73779296875, 0.42529296875, -0.7294921875, -1.9228515625, 0.303466796875, 0.043670654296875, 1.060546875, 1.26953125, -0.476318359375, 1.189453125, -0.2154541015625, -0.47314453125, -1.2177734375, 0.1346435546875, -0.054901123046875, -0.65869140625, -2.728515625, -0.19482421875, -0.7744140625, 0.071044921875, 0.46875, 0.54736328125, 0.611328125, 0.331298828125, 0.79931640625, 0.2022705078125, 0.126220703125, -1.28125, 1.154296875, -0.7041015625, 0.494873046875, 1.4658203125, -1.583984375, 0.1663818359375, -0.947265625, -0.1207275390625, 0.61376953125, -2.099609375, 1.0771484375, -0.1578369140625, 1.0703125, -0.048553466796875, -0.59375, 1.3125, 2.53515625, -0.14501953125, 0.81787109375, -0.22021484375, 0.72607421875, 0.6455078125, 0.487060546875, -0.99560546875, -1.56640625, 0.33642578125, -0.3154296875, -0.81005859375, -1.43359375, 1.150390625, -1.0185546875, 0.5849609375, 0.443359375, -1.271484375, -0.435302734375, 0.22705078125, 0.66259765625, -2.59375, -1.2158203125, -0.73095703125, 0.058868408203125, 0.318603515625, -1.8046875, -1.21484375, -1.291015625, 0.6728515625, 2.22265625, -0.80419921875, 0.27197265625, 0.19873046875, 1.767578125, 1.0830078125, -0.56591796875, -0.76953125, -0.51806640625, 1.087890625, -1.4091796875, 0.7197265625, -0.90966796875, -0.494873046875, -1.0703125, 0.062347412109375, 1.345703125, 1.8583984375, 1.263671875, 0.1739501953125, 0.428466796875, -0.367431640625, 0.42529296875, 0.08770751953125, -0.236328125, -1.2138671875, 1.01953125, 1.7529296875, -0.36181640625, -0.2186279296875, 0.7890625, -0.59326171875, 1.90625, -0.703125, -0.27197265625, -0.2159423828125, -0.12646484375, -0.252197265625, 2.0859375, -0.86962890625, -1.7158203125, 1.44140625, 0.2462158203125, -1.2783203125, 0.6201171875, -0.6982421875, -0.810546875, -0.955078125, 0.2178955078125, 0.4814453125, -0.60400390625, 0.300537109375, -2.50390625, -0.2359619140625, 0.454833984375, 1.46484375, -0.978515625, -0.8701171875, -1.7802734375, 0.27685546875, -0.251708984375, 1.59765625, -0.381591796875, -0.58740234375, -0.91162109375, 0.73388671875, -0.1575927734375, -0.15478515625, 1.1435546875, -2.3984375, 1.458984375, 0.1597900390625, -1.53515625, -0.1556396484375, 0.5224609375, 2.126953125, -1.6689453125, -0.87890625, 0.7744140625, -0.25, 0.0030422210693359375, 0.08026123046875, -0.1204833984375, -0.1776123046875, -0.87548828125, -1.451171875, -0.68408203125, 0.57373046875, 1.580078125, 0.54638671875, -0.75732421875, -0.287841796875, -0.83544921875, -0.443115234375, -0.74267578125, -1.6435546875, -0.06280517578125, -0.1488037109375, 1.642578125, 1.5, -0.73388671875, -0.51318359375, 0.283203125, 0.325927734375, -1.4716796875, -1.61328125, -0.67724609375, -0.74853515625, 0.4326171875, 0.55859375, -0.49609375, 2.138671875, 0.62158203125, 0.35009765625, -0.5537109375, -1.8232421875, 0.050201416015625, -0.161865234375, 0.286865234375, -1.7021484375, -0.857421875, -1.255859375, -0.81005859375, -0.5234375, -0.76318359375, 1.4462890625, 0.3291015625, 0.59521484375, 0.6005859375, -0.2254638671875, 1.0, 0.4013671875, 0.35546875, -1.4931640625, 0.254638671875, -0.07281494140625, 1.015625, -0.03253173828125, -0.1646728515625, -2.322265625, 1.37890625, 1.625, 0.97119140625, -1.8720703125, -0.184814453125, -0.69482421875, 1.06640625, 0.2198486328125, 0.2218017578125, 1.2568359375, 0.4365234375, -0.0220184326171875, 0.6650390625, -0.92578125, 0.1279296875, 1.1044921875, -0.94384765625, 0.423828125, 0.490966796875, 0.97119140625, 1.2724609375, -1.044921875, 1.66796875, 2.12109375, 0.314208984375, -1.37109375, -0.404052734375, -0.6669921875, -0.97509765625, -1.0498046875, -0.384033203125, 0.194091796875, -0.9208984375, 1.7333984375, -0.1383056640625, -0.00926971435546875, 0.0306243896484375, 1.765625, -1.2158203125, -0.79296875, -0.98779296875, -0.86376953125, -0.47021484375, -0.677734375, 0.744140625, -0.6875, -0.33349609375, -1.619140625, -0.87548828125, 2.146484375, -0.169921875, 1.1650390625, 1.453125, 1.435546875, -0.2169189453125, 0.6123046875, -2.25, 1.0458984375, 1.21484375, -0.361083984375, 1.0654296875, -0.47705078125, -0.623046875, -0.386962890625, -0.06182861328125, -0.888671875, 0.2044677734375, 0.60009765625, -0.304931640625, -1.2060546875, -0.91357421875, -0.9833984375, 0.08514404296875, 0.36083984375, 1.80859375, -1.30859375, -1.345703125, 2.03515625, 0.5712890625, 1.0029296875, 0.35400390625, 0.65234375, -1.3271484375, -0.0213623046875, -0.994140625, -1.513671875, 0.0175018310546875, -0.89111328125, 0.7255859375, -0.55224609375, 0.4599609375, 0.58203125, 1.1904296875, -1.5654296875, -1.41015625, 0.57666015625, 0.050048828125, 0.69189453125, -1.19921875, 0.310546875, 0.45361328125, -0.250244140625, -1.1064453125, -0.217041015625, 0.79443359375, -1.3974609375, -0.1422119140625, -0.38525390625, 0.60595703125, -0.5419921875, 1.759765625, 1.19921875, 0.81884765625, -0.079833984375, 1.3623046875, 0.455810546875, 1.2841796875, -0.546875, -1.2353515625, -0.2607421875, -1.2265625, -1.9833984375, -1.810546875, 0.440673828125, -0.057464599609375, -0.449462890625, 0.0772705078125, -0.25732421875, 1.58984375, 0.026092529296875, 0.388916015625, 0.0858154296875, 1.0634765625, 0.8173828125, 0.461181640625, -1.1796875, 0.63330078125, -1.23828125, -0.360107421875, 0.63525390625, -0.277099609375, 0.1217041015625, -0.323486328125, -0.66455078125, -2.810546875, 0.7021484375, -0.278564453125, 0.296630859375, 0.2174072265625, -0.0271148681640625, -0.381103515625, 1.611328125, 0.1983642578125, 0.6845703125, 1.318359375, 0.466064453125, 1.150390625, 0.2388916015625, -0.76123046875, 0.490234375, 0.72412109375, 0.246826171875, -0.544921875, 1.931640625, -1.060546875, -0.51123046875, 1.3544921875, 0.33544921875, -0.6259765625, 2.279296875, -0.064208984375, 1.017578125, -2.232421875, 0.7021484375, -0.381591796875, 1.9765625, 2.873046875, -0.76025390625, 0.59326171875, 1.25390625, 1.7822265625, 0.24072265625, -0.82177734375, -4.18359375, 0.07452392578125, -0.251953125, 0.1295166015625, -2.45703125, -0.37548828125, -0.497314453125, 1.224609375, -2.416015625, -0.39013671875, 0.046417236328125, 0.60205078125, 0.367919921875, 0.432373046875, -0.3720703125, 2.4296875, -2.248046875, 0.35498046875, 0.56640625, 0.493896484375, 0.26708984375, 0.53955078125, -0.89013671875, -2.064453125, -1.3310546875, 0.765625, 0.68310546875, 0.05987548828125, -1.5107421875, 1.2783203125, -0.0248565673828125, 0.056182861328125, 2.080078125, -0.94287109375, -0.16796875, -0.11065673828125, 1.4072265625, 0.275634765625, 0.82861328125, 0.81591796875, 0.333251953125, 0.37841796875, -0.76171875, 1.04296875, 0.37158203125, 0.501953125, -1.4580078125, 1.9794921875, -0.442138671875, 1.205078125, -1.4873046875, -0.6142578125, -2.09765625, -0.124267578125, -0.7666015625, 0.50341796875, -1.447265625, -0.431396484375, -0.787109375, 0.8544921875, 0.328857421875, -0.7333984375, -0.5849609375, -1.1875, 0.302978515625, -1.791015625, 0.35009765625, 1.9501953125, -0.64453125, -2.287109375, 0.0389404296875, -1.4580078125, 2.314453125, -0.58740234375, 0.56494140625, -0.53564453125, -0.7373046875, -0.6240234375, -0.86474609375, -0.69677734375, 0.25927734375, -0.56982421875, -1.9306640625, 0.15869140625, -2.044921875, 1.3779296875, -0.5390625, 0.2320556640625, 0.8955078125, 1.4072265625, -0.60107421875, 0.318603515625, -0.7626953125, 0.338623046875, -0.230224609375, -0.233154296875, 0.9267578125, 1.501953125, 0.91845703125, 0.556640625, 0.9404296875, 0.2496337890625, 0.41357421875, -0.5, -0.62353515625, -0.208740234375, 0.80859375, -1.32421875, -0.42138671875, -0.163330078125, -1.28125, -0.599609375, -1.4833984375, -0.77734375, -0.86083984375, 1.041015625, 0.48876953125, -0.2454833984375, 1.0859375, 0.86669921875, -2.9140625, -0.8876953125, -0.10150146484375, 0.70654296875, 1.0458984375, -0.7451171875, -1.060546875, 0.9697265625, 1.1240234375, -1.3271484375, -0.83740234375, -0.935546875, -0.58837890625, 0.022216796875, -2.294921875, -0.9580078125, -0.53564453125, -0.496826171875, 1.56640625, -1.5888671875, -0.0252685546875, 1.212890625, -0.857421875, -0.640625, 0.40087890625, 1.1025390625, -1.2421875, 1.4384765625, -0.278076171875, -0.051727294921875, -0.8330078125, 0.9375, -0.255859375, -0.40478515625, 0.63134765625, 0.25830078125, -2.0703125, -1.4873046875, -0.9775390625, 0.11614990234375, 0.55029296875, -0.92578125, 0.12066650390625, 2.015625, 0.693359375, 1.001953125, 0.66796875, -0.9921875, 1.8525390625, -0.448486328125, -1.158203125, 0.2305908203125, -0.75732421875, 0.9287109375, -0.1446533203125, -1.9091796875, 1.8076171875, 1.94921875, 0.71240234375, -0.043304443359375, 0.258544921875, 2.3671875, 0.720703125, 1.6953125, 0.65185546875, -0.455810546875, 0.07171630859375, 0.732421875, -0.374267578125, -2.3359375, 0.24462890625, -0.822265625, -1.6552734375, -0.09161376953125, -0.8896484375, -1.3486328125, 1.876953125, -0.99609375, 0.0198516845703125, 1.1904296875, 1.69140625, -1.54296875, 0.4755859375, -1.9658203125, -0.31201171875, 0.278564453125, 0.0176239013671875, 0.486083984375, 0.01013946533203125, 0.234619140625, 0.8095703125, -0.34033203125, 1.458984375, -2.216796875, -1.669921875, 0.223876953125, -0.5302734375, -1.470703125, 2.2734375, 0.59033203125, -0.5986328125, -0.93115234375, -0.63037109375, 1.6123046875, 2.265625, -0.7138671875, 0.137451171875, -0.01340484619140625, -0.671875, 0.98388671875, 0.0264129638671875, 1.333984375, -0.1148681640625, -0.97119140625, 0.2010498046875, -0.857421875, 1.0947265625, -1.130859375, -1.4208984375, 0.658203125, 0.2890625, 1.361328125, -0.7451171875, -2.0078125, 0.136962890625, -0.54296875, -0.80078125, -1.388671875, 0.086181640625, 0.1932373046875, 0.297607421875, 0.05804443359375, -0.52978515625, 0.77490234375, 0.5224609375, 1.25390625, 0.96484375, -1.439453125, 0.69775390625, 0.25341796875, 1.46484375, 1.3466796875, -0.011810302734375, -1.841796875, 0.0029163360595703125, -1.02734375, -0.8154296875, -0.98095703125, 0.83642578125, -0.040313720703125, 1.11328125, 1.966796875, -0.0173492431640625, -0.86376953125, -0.90234375, 0.9443359375, 0.328369140625, 0.6513671875, -1.1826171875, -0.64404296875, 0.6201171875, -1.9541015625, 0.54638671875, 0.345947265625, -1.1611328125, 0.46875, -0.32373046875, 2.87890625, -1.072265625, 0.9130859375, -0.96484375, -0.0194549560546875, 0.48486328125, 1.1337890625, 0.1876220703125, 0.142822265625, -0.18212890625, -2.1953125, 1.5595703125, 0.1759033203125, 0.3388671875, 2.3125, -1.90234375, -0.75732421875, 0.371826171875, -0.91015625, -0.86181640625, 0.79052734375, 1.0068359375, 1.5927734375, -0.059356689453125, -1.9951171875, 1.189453125, -1.4169921875, -0.26416015625, 1.1845703125, -0.73095703125, 0.174560546875, -1.0009765625, -0.92333984375, -0.033203125, 1.22265625, 1.423828125, -0.380615234375, -2.09375, 0.0250091552734375, -1.189453125, 0.288330078125, 0.12286376953125, 0.12359619140625, 1.48046875, 1.166015625, -0.72705078125, -0.66455078125, -0.361083984375, -0.759765625, -0.2369384765625, -0.9423828125, 0.96484375, 0.6123046875, -1.576171875, -0.009796142578125, 0.60009765625, -0.315185546875, 0.97412109375, 0.830078125, -0.2041015625, 0.36865234375, 2.236328125, -0.12164306640625, 0.178466796875, 0.365234375, 0.97021484375, 0.25146484375, -0.05462646484375, -0.23193359375, 1.158203125, 0.50634765625, 0.1212158203125, -0.9404296875, -1.392578125, 0.52392578125, -0.2120361328125, -0.74755859375, 0.28076171875, 1.0673828125, -0.275146484375, 0.70947265625, -0.89697265625, -1.0771484375, 1.1484375, 0.37109375, -1.693359375, -0.10247802734375, 2.65234375, -0.4814453125, 0.4111328125, 0.1070556640625, 0.806640625, -0.162841796875, 1.71484375, 1.8212890625, 0.68359375, -1.0693359375, -0.71484375, 0.005603790283203125, 0.99267578125, 1.2216796875, -1.603515625, 0.360595703125, -0.09295654296875, -0.154052734375, 0.413330078125, 0.450927734375, 0.587890625, 1.1171875, -0.89453125, 0.5986328125, -1.3076171875, 0.0279541015625, -0.4560546875, 1.6474609375, -0.97021484375, -0.3291015625, -0.8671875, -1.00390625, 0.47607421875, 0.91552734375, 1.673828125, -1.7626953125, 0.4912109375, 1.37890625, 0.94287109375, -0.292724609375, 0.1488037109375, -0.07025146484375, -1.1962890625, -0.3134765625, -1.3896484375, 0.410400390625, -0.057403564453125, -1.2626953125, -0.33642578125, -1.5341796875, -0.40576171875, 1.1240234375, -0.469482421875, -1.177734375, 1.234375, -0.490234375, 1.048828125, 0.493408203125, -0.07080078125, -0.82958984375, -0.1075439453125, 2.6484375, -1.2001953125, 1.001953125, 1.068359375, 0.92529296875, 1.400390625, 2.169921875, 0.336181640625, -1.1298828125, 0.89697265625, -2.025390625, -0.484619140625, 1.0205078125, 1.5390625, -1.2763671875, 1.248046875, -0.26318359375, 0.91064453125, -0.27880859375, 0.348876953125, -0.11163330078125, 1.9013671875, -1.1103515625, -1.1201171875, 1.3154296875, 0.615234375, -2.21875, -2.470703125, 0.6884765625, -0.97021484375, 0.43359375, 1.0615234375, -0.364501953125, 2.3984375, 0.96240234375, -0.1346435546875, -0.05828857421875, -0.3486328125, -0.7470703125, -1.2451171875, -0.271240234375, 0.740234375, -0.08856201171875, -0.78125, -1.40234375, 0.306640625, 0.53125, 1.4580078125, 1.283203125, -2.3046875, 1.4208984375, -1.482421875, 1.23828125, 0.44091796875, 0.443115234375, -0.2315673828125, 0.310302734375, -0.61572265625, 1.0234375, 1.3623046875, 1.4365234375, 0.06451416015625, 1.521484375, -0.73046875, -1.1357421875, -1.4453125, 1.6484375, 0.68017578125, 0.226806640625, -0.369384765625, 0.9326171875, 0.01091766357421875, 1.65234375, -1.1396484375, 1.021484375, 0.7158203125, 1.6201171875, -0.46826171875, 0.0213470458984375, -1.9453125, -0.251708984375, 0.205322265625, -0.429443359375, 1.0986328125, 0.412353515625, 0.85009765625, 2.287109375, 0.80615234375, 1.0380859375, 0.34130859375, -0.57568359375, -1.19140625, 0.40087890625, 0.830078125, 0.034423828125, -0.513671875, 0.27734375, -1.42578125, -1.66015625, 1.1728515625, -0.279296875, 0.36083984375, 0.75048828125, 1.544921875, 1.62890625, 0.272705078125, -0.1593017578125, -0.9501953125, -0.56591796875, -2.3203125, 0.3994140625, -0.450439453125, -1.3837890625, 0.46044921875, 1.736328125, 1.0087890625, 2.05078125, 1.37890625, -1.5341796875, 0.09039306640625, -0.67626953125, -2.1171875, 0.39453125, 0.92529296875, 1.6572265625, -0.235107421875, -0.3115234375, 0.56494140625, 1.6669921875, 0.6708984375, 1.80078125, 0.8896484375, 0.603515625, -0.1605224609375, 0.859375, -0.8955078125, -0.156982421875, -0.763671875, -1.78125, -0.5009765625, -0.8203125, -0.454833984375, 2.427734375, -0.345947265625, -0.328125, -0.1407470703125, -1.0576171875, 1.484375, -0.168701171875, -0.6826171875, 1.8623046875, 0.32861328125, 0.37109375, -0.93212890625, -0.0265655517578125, 0.62158203125, -0.80322265625, 0.33349609375, -0.701171875, 0.4833984375, 0.044189453125, -0.775390625, 1.705078125, 1.4130859375, 0.9111328125, -1.5849609375, 0.007602691650390625, -0.483642578125, 1.2451171875, -2.287109375, -1.373046875, -1.3310546875, -0.970703125, -0.146728515625, 0.623046875, -0.024139404296875, -0.98291015625, -0.8056640625, 1.275390625, -0.312744140625, 0.98486328125, 0.141845703125, 0.1024169921875, -1.5556640625, -1.3623046875, 1.248046875, 0.759765625, 1.5693359375, -0.7626953125, 0.673828125, -0.474853515625, 0.52392578125, 1.3642578125, -0.9560546875, 1.0244140625, 0.89697265625, -0.404296875, -0.30517578125, -0.525390625, -1.755859375, 0.1754150390625, 1.294921875, -0.9541015625, -0.8125, 1.5087890625, 0.83642578125, 0.73291015625, -0.242431640625, 1.1201171875, 0.57177734375, -0.18505859375, -0.08441162109375, 1.34375, 1.25, 0.74853515625, 2.796875, 1.259765625, -0.623046875, 1.224609375, 0.283935546875, 0.481201171875, 0.97509765625, 1.162109375, 0.2337646484375, -0.93701171875, -1.9267578125, -0.141845703125, 0.5869140625, 0.39697265625, 0.70068359375, -0.0482177734375, 1.119140625, -0.33544921875, -0.2626953125, 1.373046875, -0.73046875, 2.208984375, 0.11651611328125, 1.7431640625, 0.75390625, 0.259521484375, -0.361328125, 0.09375, 0.11187744140625, -0.63720703125, -0.11798095703125, 0.0280914306640625, -0.2347412109375, -0.466064453125, 0.032135009765625, 0.5107421875, -1.7041015625, -1.255859375, -0.552734375, -1.32421875, -0.25390625, 1.4794921875, 0.30517578125, 0.447021484375, 1.5244140625, 1.25, -0.81982421875, 1.125, 0.53662109375, 0.08612060546875, -0.7255859375, -0.65087890625, -0.347900390625, 0.8681640625, -0.271240234375, 0.061614990234375, -0.01079559326171875, 0.6279296875, -0.1815185546875, -0.23388671875, -0.1229248046875, 0.2626953125, 0.38232421875, 0.4658203125, -1.2802734375, 0.295166015625, 0.05841064453125, -0.833984375, -0.10614013671875, -0.22021484375, 0.0704345703125, -1.0224609375, 0.1480712890625, -0.63623046875, -2.484375, -0.755859375, -0.271240234375, -0.07177734375, 0.826171875, -0.043426513671875, -0.0650634765625, 0.30322265625, 0.21484375, -0.658203125, -1.185546875, 1.123046875, -1.0439453125, 0.0928955078125, -1.2548828125, -0.33984375, 1.2509765625, 0.50732421875, 0.1339111328125, 0.96337890625, 0.2344970703125, -1.2978515625, 1.978515625, 0.432373046875, 0.744140625, -0.5302734375, -0.80419921875, 0.09576416015625, 1.208984375, -0.904296875, -0.90234375, -0.66748046875, 0.4775390625, 2.556640625, -0.7587890625, 0.250732421875, 1.0322265625, -0.4521484375, 0.0262298583984375, -2.08984375, -0.646484375, -1.673828125, 1.5400390625, 2.015625, 1.1591796875, -0.8720703125, -0.90576171875, 1.685546875, -0.73095703125, 2.51171875, 0.5419921875, 0.85888671875, 1.033203125, -0.4775390625, -0.61669921875, 0.093017578125, 0.71533203125, -0.23486328125, -1.1318359375, -3.30859375, -1.326171875, 0.732421875, -0.4521484375, 2.078125, -0.12744140625, 0.1370849609375, -1.4658203125, 0.271484375, 0.1524658203125, 0.1834716796875, -1.60546875, 0.47119140625, -0.058380126953125, 0.59326171875, 0.125244140625, 0.79052734375, 0.52783203125, 0.07684326171875, -0.57666015625, -0.88134765625, 0.72998046875, -0.42529296875, 1.09375, 0.28662109375, -0.50927734375, 0.365234375, -0.05621337890625, 0.52099609375, 0.81298828125, -0.52392578125, -1.25390625, -0.57373046875, -0.9892578125, -1.080078125, -0.0731201171875, -0.250244140625, 1.365234375, -0.68798828125, 1.0234375, -0.21484375, -0.73828125, 1.6708984375, -1.453125, -0.136962890625, -0.10406494140625, -0.513671875, -0.46435546875, -2.63671875, -1.2822265625, 0.348388671875, 1.7314453125, 1.654296875, 0.045806884765625, 0.1304931640625, -0.3681640625, -0.89794921875, -2.814453125, 0.56396484375, 0.890625, -0.916015625, -0.408447265625, -0.3544921875, -0.82470703125, 0.56787109375, 1.7509765625, -0.8681640625, 0.376220703125, 0.1925048828125, -0.1036376953125, -0.29833984375, -0.708984375, -0.06732177734375, 0.63671875, 0.228759765625, 0.55322265625, -1.3076171875, 2.009765625, -0.355224609375, -0.311767578125, -0.326904296875, 2.16796875, 0.28076171875, -0.040740966796875, -1.484375, 0.0298004150390625, 1.546875, 0.57080078125, 2.02734375, -1.310546875, -0.7490234375, -1.923828125, 0.5068359375, 2.326171875, -0.484130859375, -2.798828125, 0.52099609375, -1.763671875, -1.4033203125, 1.2919921875, 0.70947265625, 1.08984375, 0.306640625, -0.40380859375, -0.80224609375, 1.13671875, 1.552734375, -0.0119476318359375, 1.650390625, 0.99658203125, -0.77734375, -0.66845703125, -0.2464599609375, -0.970703125, -0.9638671875, -2.4921875, -1.2744140625, -0.68505859375, 1.642578125, 0.90087890625, -2.119140625, -1.4716796875, -1.2119140625, 0.11578369140625, 0.1121826171875, 1.2568359375, 0.744140625, -1.83984375, 0.0130462646484375, -0.25146484375, 0.8662109375, 0.150634765625, -0.53125, 1.1435546875, -0.57470703125, 0.240478515625, 1.05078125, -2.62109375, 0.50927734375, -0.92041015625, -0.045318603515625, 0.740234375, -0.67724609375, -0.37353515625, 1.2158203125, 0.2303466796875, 0.3173828125, -0.626953125, 0.970703125, 0.1680908203125, -1.4501953125, 0.267822265625, 1.58984375, -1.6552734375, 0.87109375, -0.7060546875, 1.435546875, -0.73046875, -0.24267578125, -0.79052734375, -0.634765625, -0.15869140625, 0.05010986328125, -1.2919921875, 1.193359375, 0.83154296875, 0.46240234375, 0.666015625, 0.78125, 0.2958984375, -0.43408203125, 1.4775390625, 0.2354736328125, 1.8232421875, 1.736328125, -0.68505859375, -1.3681640625, 0.91455078125, -0.345703125, -0.7626953125, -0.048980712890625, 0.278076171875, 0.6845703125, 0.5517578125, -0.984375, 0.6015625, 0.57763671875, -0.412353515625, 1.4033203125, 1.0302734375, 1.42578125, 1.0859375, -0.436279296875, 2.890625, 0.763671875, 1.712890625, -0.60009765625, 0.7109375, 0.51123046875, -0.859375, -0.9609375, -0.7021484375, 0.006290435791015625, 0.31982421875, 0.61962890625, 0.259521484375, 0.51953125, 0.78662109375, 1.10546875, -1.2119140625, -1.490234375, 0.294921875, 1.5576171875, -0.86083984375, -0.93212890625, 0.2529296875, 0.28125, -0.51171875, 0.28271484375, -0.43994140625, -0.330810546875, 0.779296875, -0.564453125, 0.1253662109375, 0.06475830078125, 0.4072265625, -1.22265625, -0.798828125, -0.8818359375, -0.9873046875, -0.3564453125, -0.08990478515625, -1.4599609375, 1.2333984375, -0.275634765625, 0.00704193115234375, 3.73046875, 0.3193359375, 0.2435302734375, -0.68017578125, -1.2900390625, 0.69921875, 0.44091796875, -1.498046875, 0.0095062255859375, 1.0517578125, -0.46337890625, -0.7783203125, -0.15185546875, 1.4013671875, -0.01186370849609375, -0.6298828125, -0.787109375, 0.525390625, -0.035797119140625, -0.201416015625, 0.436767578125, -0.426513671875, 0.8798828125, 0.343017578125, -0.54345703125, 1.6982421875, -0.62451171875, 0.93505859375, 0.20068359375, 0.415283203125, -1.0751953125, 1.0361328125, -0.065673828125, -1.8828125, 0.65625, -0.9990234375, 0.415283203125, -0.49267578125, 1.74609375, 1.021484375, 1.0947265625, 0.64599609375, 0.92919921875, -0.6171875, 0.58203125, -0.1112060546875, -2.775390625, 0.2049560546875, -1.265625, -1.3916015625, 0.5078125, -0.2021484375, 0.2366943359375, 2.240234375, -0.962890625, -1.12109375, 0.86376953125, -0.486572265625, -0.021026611328125, -0.039093017578125, -1.1474609375, 0.72314453125, 1.0302734375, -1.7265625, -0.73828125, -1.2470703125, -1.1767578125, -0.7548828125, 0.5859375, 0.434814453125, -1.01953125, 0.673828125, 0.1505126953125, -2.0859375, 0.107177734375, 2.013671875, -0.27587890625, 0.7373046875, -0.666015625, 1.0615234375, -0.7763671875, -2.578125, -1.1171875, -1.4091796875, 0.0128173828125, -1.251953125, -0.47900390625, -1.8076171875, 0.83251953125, 1.1435546875, -0.564453125, -1.5556640625, 0.984375, -0.087158203125, 0.958984375, 0.351806640625, -1.2109375, -0.10205078125, -0.57958984375, -1.0634765625, 0.5966796875, 1.3349609375, -0.2100830078125, 0.59716796875, 0.1502685546875, -1.4384765625, -0.3232421875, -1.5087890625, -0.5625, -1.6328125, 0.443115234375, -1.08203125, 0.93603515625, -0.2998046875, -0.50732421875, -1.142578125, 0.7333984375, 0.515625, -2.34765625, 0.83154296875, -0.11279296875, 0.420166015625, 0.9912109375, 0.28564453125, -0.9501953125, -0.8115234375, -0.97314453125, -1.2548828125, 0.246337890625, -0.73193359375, 0.6005859375, -0.2469482421875, -0.0760498046875, -2.076171875, 0.66552734375, 0.111083984375, -0.02947998046875, -0.07733154296875, -1.76171875, 1.33984375, -0.301025390625, 0.5830078125, 0.375, 0.0601806640625, -0.6435546875, 0.385498046875, -0.205078125, -0.9765625, -1.8935546875, -0.51416015625, 0.0692138671875, -1.3759765625, 0.0355224609375, 0.11151123046875, 0.8369140625, 2.6484375, 1.6142578125, 1.33984375, 0.6552734375, -0.00414276123046875, 0.4609375, -1.4931640625, 1.98828125, 0.357666015625, -0.88916015625, 1.8671875, 0.87841796875, 0.4482421875, 0.78125, 0.42333984375, -1.001953125, -0.76513671875, 0.177734375, -0.279296875, -0.50537109375, 0.139404296875, -0.63671875, -0.179931640625, 1.24609375, -0.962890625, -1.9033203125, -0.7109375, -1.2763671875, 1.8583984375, -0.08837890625, -0.59521484375, -0.0955810546875, 1.138671875, -0.0275726318359375, 1.216796875, -0.9033203125, 1.2919921875, -0.63427734375, -0.1458740234375, 0.413330078125, 0.48095703125, -0.74169921875, 2.146484375, -0.362548828125, 1.6279296875, 0.096923828125, -1.4013671875, -0.37548828125, 0.912109375, -0.62158203125, -0.421630859375, -0.0241546630859375, -1.322265625, 1.615234375, -1.9423828125, -0.52197265625, -1.59375, 0.62841796875, -0.27490234375, -0.89599609375, 0.81298828125, 0.3623046875, 0.24560546875, 2.37109375, -0.9814453125, -0.5908203125, 0.63525390625, 0.453369140625, -0.01220703125, 0.11602783203125, -0.66943359375, -0.03900146484375, -0.2420654296875, 0.51513671875, 0.771484375, 2.44140625, -1.4384765625, 1.224609375, 0.322265625, 1.7294921875, -0.032257080078125, 1.58203125, -0.070068359375, 0.02093505859375, -0.89013671875, -0.03338623046875, 0.943359375, 0.173828125, -0.73974609375, -0.1527099609375, -0.44580078125, 0.775390625, 0.0213165283203125, 0.51416015625, 0.71337890625, -1.6044921875, -0.1600341796875, -0.4169921875, -1.9765625, 0.65576171875, -1.54296875, 0.2269287109375, -1.0263671875, 1.2109375, 1.6708984375, -0.276611328125, 0.82421875, -1.2412109375, 0.6455078125, 1.0419921875, 0.5927734375, 0.265380859375, 0.029144287109375, -1.3779296875, -1.2978515625, -0.6416015625, -0.66015625, -0.8134765625, 0.481201171875, 0.061187744140625, 0.36181640625, 1.0537109375, -0.08856201171875, -0.51611328125, -1.10546875, -0.6171875, -0.421875, 0.72998046875, 0.6591796875, 0.178466796875, -1.22265625, -0.8076171875, 2.009765625, -0.580078125, 0.372802734375, 0.08050537109375, -2.66015625, -0.73193359375, -0.390869140625, -0.685546875, -0.27734375, 1.37890625, 0.200439453125, 1.24609375, -1.0849609375, 0.89697265625, -0.81591796875, 0.2401123046875, -0.896484375, 1.611328125, 0.607421875, -0.32275390625, -0.4560546875, 0.388671875, -0.296875, 0.1590576171875, 0.443603515625, 1.984375, -0.58251953125, -1.7431640625, 2.626953125, -0.908203125, 0.865234375, -0.662109375, -0.1513671875, -0.00789642333984375, -0.2724609375, 1.40625, 0.11468505859375, 0.492431640625, -1.6025390625, 2.583984375, 1.552734375, 0.904296875, -0.63916015625, -0.055511474609375, -0.87841796875, -0.0232696533203125, -0.68017578125, 1.1318359375, -0.11181640625, -1.2802734375, 0.35791015625, 0.1612548828125, -1.51953125, -0.55419921875, 0.476318359375, -0.3759765625, 0.1534423828125, -3.814453125, -1.4599609375, -1.5830078125, 0.841796875, 0.1763916015625, 0.01117706298828125, 0.0196380615234375, -0.237060546875, 0.3203125, -0.45947265625, 0.90966796875, -2.072265625, -1.1064453125, -1.1103515625, -0.83935546875, -0.92236328125, -1.576171875, 1.1689453125, 1.84765625, 1.4580078125, 0.074462890625, -1.078125, -0.4775390625, -3.1328125, -0.83935546875, -0.363037109375, -0.09503173828125, -0.61767578125, 1.4326171875, -0.300537109375, -2.189453125, 2.146484375, -2.078125, -0.31982421875, -0.78125, -0.06097412109375, 0.1536865234375, -0.41552734375, -0.135009765625, 0.73583984375, 2.669921875, 0.17236328125, -1.103515625, 0.0745849609375, -2.0390625, 1.759765625, 0.875, 1.3173828125, 0.046875, -0.251708984375, -1.05859375, 0.73388671875, -0.43310546875, -0.47265625, -0.83740234375, -0.399169921875, 0.66455078125, -0.412353515625, -0.32763671875, 1.7470703125, -0.2861328125, -0.53369140625, 0.5458984375, 1.318359375, -1.6767578125, -0.935546875, 0.79248046875, 0.68359375, 0.0836181640625, -0.1318359375, -0.050445556640625, -0.9609375, 0.057373046875, -1.451171875, 1.7802734375, 0.98828125, 1.16796875, 0.11578369140625, 0.591796875, -0.55810546875, -0.85546875, 0.6376953125, -0.796875, 0.37451171875, -0.2320556640625, -1.3173828125, -0.73046875, -1.7294921875, 0.64013671875, 0.693359375, 0.9521484375, 1.5576171875, 1.2333984375, -1.3505859375, -0.86328125, -0.324951171875, -1.365234375, -0.0257110595703125, 0.6298828125, -1.48828125, -2.34375, 0.0272979736328125, 0.78466796875, 1.1103515625, -0.034393310546875, -0.328369140625, -0.07659912109375, -1.00390625, 1.4384765625, 0.97021484375, -1.3583984375, -0.80517578125, -1.36328125, -1.0380859375, -1.3046875, 1.267578125, 0.10321044921875, -1.1435546875, -0.28564453125, -2.77734375, -0.58837890625, 0.325439453125, 1.1513671875, -0.64892578125, 0.91943359375, 1.4248046875, -0.650390625, -0.268798828125, -0.1729736328125, -0.6611328125, -0.4677734375, -0.0242919921875, -0.30859375, -0.4140625, -0.560546875, 0.75146484375, -0.446533203125, 2.65234375, -0.62109375, -0.73388671875, -1.3095703125, -0.361572265625, 1.8349609375, 0.03692626953125, 0.951171875, -1.0810546875, -1.66015625, 1.21875, 0.05926513671875, -0.414794921875, -2.8984375, 1.0244140625, 0.97509765625, 0.212158203125, 1.8046875, -2.15234375, 0.267822265625, -0.9462890625, 0.218994140625, 0.884765625, 0.671875, -1.5029296875, 0.671875, -0.2015380859375, 2.58203125, 2.439453125, 0.10687255859375, 0.62109375, 0.056243896484375, 0.59228515625, 0.712890625, -1.2666015625, -0.5234375, -0.1866455078125, 1.7587890625, 0.72705078125, 0.4365234375, -0.19140625, -2.06640625, 0.090087890625, -2.03515625, 0.3662109375, -2.41015625, 2.259765625, -0.32373046875, -1.0087890625, 1.1904296875, -0.65673828125, 0.88916015625, 0.2529296875, 0.46435546875, 1.291015625, -0.8466796875, 0.6923828125, 0.171630859375, 0.6669921875, -0.94482421875, 1.4697265625, 2.037109375, 0.048187255859375, 1.869140625, 0.98583984375, -0.89453125, 0.37548828125, 0.77734375, -1.4560546875, -1.6953125, 0.615234375, 0.403564453125, -0.55126953125, 0.464599609375, 0.394287109375, 0.86083984375, 1.7109375, -0.17919921875, -3.6796875, 0.2386474609375, -0.5947265625, -0.44384765625, 0.4296875, 0.287841796875, 1.6171875, -0.4521484375, -0.4365234375, 2.234375, -1.2041015625, -0.50732421875, -0.73193359375, 1.76171875, 0.292236328125, 1.6162109375, -0.8291015625, 0.04962158203125, -0.47314453125, -0.172119140625, -0.2081298828125, 0.221923828125, 1.5166015625, -0.263916015625, -1.3193359375, -1.0947265625, -0.25048828125, -0.08624267578125, 1.490234375, 0.281005859375, 0.8916015625, 0.3046875, -0.537109375, 1.6953125, 2.029296875, -0.84375, -1.025390625, 0.172607421875, -0.431396484375, -1.1455078125, 1.0576171875, 1.474609375, -0.517578125, 1.25390625, -0.595703125, -2.048828125, 0.057098388671875, -0.344482421875, 0.35302734375, 1.0322265625, -1.6357421875, 1.5458984375, -0.7919921875, 0.93212890625, -0.319091796875, 0.66552734375, 0.91650390625, -1.701171875, -0.8076171875, -0.1168212890625, -1.3046875, -0.1513671875, -1.111328125, 0.5341796875, -1.0244140625, -0.88525390625, 1.1318359375, -1.01171875, -0.58935546875, 1.6669921875, 1.595703125, -0.99951171875, 0.93701171875, 1.7470703125, 0.0966796875, 0.94580078125, -0.395751953125, 0.35205078125, -0.87890625, -0.50146484375, 1.595703125, -1.2080078125, -2.103515625, -0.57958984375, -0.87646484375, -0.32470703125, 1.5380859375, 0.43505859375, 0.4033203125, 0.04046630859375, -2.49609375, 1.12109375, 0.89501953125, -0.5625, 0.83984375, 0.34033203125, 1.0947265625, -1.9365234375, -3.326171875, -0.71875, 0.8134765625, -0.191650390625, -0.099365234375, -1.541015625, 2.375, -0.466552734375, 0.0828857421875, -1.1552734375, -0.5302734375, 0.97998046875, 0.9697265625, 0.435302734375, 0.56591796875, 1.3037109375, 0.486572265625, 0.1746826171875, 1.1884765625, 0.324951171875, -0.385986328125, -0.572265625, -0.394287109375, -0.13525390625, -1.3564453125, -1.14453125, 0.8583984375, -0.3212890625, -0.2347412109375, 2.34375, -1.0341796875, 0.090576171875, 0.97705078125, -0.10650634765625, -0.64892578125, 1.4130859375, 1.2861328125, -0.29833984375, 0.229248046875, -0.2264404296875, 0.1170654296875, -0.425537109375, 0.51904296875, -1.4208984375, -0.52783203125, -0.294921875, -0.97705078125, 1.1767578125, 0.857421875, -1.4423828125, 0.9658203125, -0.1004638671875, -0.74951171875, -1.0458984375, -0.50537109375, -0.62353515625, 0.50341796875, -1.255859375, -1.1611328125, -0.6728515625, 0.90283203125, 0.0101165771484375, -0.6689453125, -0.82421875, 0.73779296875, -0.061187744140625, 0.86669921875, -0.5341796875, 1.6943359375, 1.0712890625, -1.0234375, -1.494140625, -0.6064453125, -1.2841796875, -0.130615234375, 0.2379150390625, -0.90966796875, 1.953125, 0.80078125, -0.37158203125, 2.5390625, -0.2464599609375, -1.078125, 1.2255859375, 1.240234375, -0.286376953125, -1.3720703125, -0.6533203125, -0.28857421875, -0.327880859375, -0.81298828125, 1.908203125, 0.40673828125, 1.234375, -0.446533203125, 0.6826171875, -0.0889892578125, 0.198974609375, -0.09912109375, 0.64306640625, -0.342041015625, -0.7451171875, 0.7783203125, -3.330078125, 2.240234375, -1.357421875, -1.59765625, -0.9150390625, -1.62109375, -0.7685546875, 0.59912109375, 0.7216796875, -0.215087890625, -2.158203125, -0.69580078125, -0.61181640625, 0.84619140625, 2.029296875, 0.08489990234375, 1.48046875, 0.8154296875, 0.315673828125, 0.130615234375, -0.8916015625, -1.861328125, -0.41162109375, 0.4150390625, 0.736328125, -0.8232421875, 0.5869140625, -0.81689453125, 0.7646484375, -0.48388671875, 0.67236328125, -1.724609375, 1.1982421875, 0.7919921875, -0.609375, -0.22314453125, -1.2646484375, 2.03515625, 0.85791015625, -0.39794921875, 0.50830078125, 2.443359375, -1.005859375, -1.2451171875, -1.5869140625, -0.0012950897216796875, -0.91357421875, -2.21875, -1.896484375, -0.4794921875, -1.6103515625, -0.356689453125, 1.8564453125, -1.1142578125, 0.49755859375, -1.6875, -1.2939453125, -1.171875, 1.42578125, 0.7041015625, -1.4072265625, 0.478271484375, 0.47509765625, 1.0771484375, -0.2734375, -0.08575439453125, 0.77490234375, 0.016387939453125, 0.40478515625, -0.27587890625, 0.271484375, -1.5791015625, 1.419921875, -0.494873046875, 0.2149658203125, 0.4921875, 1.08203125, 1.0732421875, 0.409423828125, 0.9306640625, 0.25146484375, 0.449951171875, 1.5185546875, -0.2305908203125, 0.802734375, -0.431640625, 0.77880859375, -1.3466796875, -1.0966796875, -0.623046875, 0.267333984375, 0.357177734375, -0.51513671875, -0.07843017578125, 0.288330078125, 1.06640625, 0.5263671875, 1.59375, -0.5068359375, 1.8203125, -0.619140625, -0.44775390625, 0.259033203125, -1.1552734375, -1.75, -0.537109375, -0.51123046875, -1.0654296875, -0.094482421875, 0.2734375, -0.650390625, -0.369140625, 1.03125, -1.6494140625, -0.677734375, 0.90869140625, -0.61962890625, 0.318603515625, -1.294921875, -0.45458984375, 0.1536865234375, -0.41357421875, -0.82666015625, -1.509765625, 0.5537109375, -0.4287109375, -0.343505859375, -2.2109375, -0.791015625, 0.463623046875, -0.69287109375, -1.8056640625, -0.76513671875, 1.455078125, -0.053314208984375, -0.4658203125, 0.1051025390625, 0.41748046875, 0.8310546875, -0.35791015625, -0.288818359375, 0.477783203125, 0.038482666015625, 0.81298828125, 1.4267578125, -1.56640625, 0.276611328125, -0.7275390625, -0.0243377685546875, -2.48828125, -0.007122039794921875, 0.10235595703125, 1.3017578125, 0.1202392578125, 0.5732421875, -1.8037109375, 0.396484375, -0.79443359375, 1.3837890625, -1.3203125, -1.1474609375, -0.462158203125, -0.29150390625, -1.6162109375, 0.60693359375, -0.52294921875, 2.138671875, -0.35546875, -1.357421875, -1.400390625, -0.39892578125, 0.671875, 0.26953125, 1.0361328125, -1.3173828125, -0.1875, 1.0498046875, 1.138671875, -0.71484375, -0.552734375, 0.0875244140625, -0.8388671875, -0.09747314453125, 1.15625, 0.52734375, 1.48046875, 0.484619140625, 0.444580078125, -0.0290679931640625, 1.951171875, 0.99365234375, 0.034423828125, -0.50341796875, 0.445068359375, 0.2320556640625, -1.255859375, -0.4501953125, -1.025390625, -0.2105712890625, 0.1470947265625, 2.51953125, -0.1658935546875, -1.09765625, 0.0848388671875, -0.54150390625, 1.4248046875, -1.5380859375, -1.1533203125, 0.0411376953125, -1.7900390625, -0.13818359375, 0.2900390625, -0.185791015625, 0.17724609375, -0.8232421875, 1.3330078125, 2.416015625, 0.467041015625, -0.1029052734375, 0.53076171875, -0.07080078125, -0.671875, -0.47705078125, 0.72314453125, -0.307861328125, -0.2412109375, -0.10821533203125, 2.884765625, -0.9189453125, -0.00545501708984375, -0.6708984375, 1.609375, -0.3310546875, -0.447998046875, 0.88916015625, -0.1361083984375, 1.947265625, 0.1224365234375, 0.0657958984375, 2.1015625, 1.1513671875, -0.59423828125, -0.08514404296875, 0.142333984375, 0.58154296875, -0.70947265625, 0.63916015625, -1.0703125, -0.242919921875, -0.1966552734375, 0.247314453125, -1.7314453125, 1.9375, -0.91552734375, -0.017059326171875, 0.9765625, 1.3203125, 0.81103515625, 1.4150390625, -1.3681640625, -0.355712890625, 0.5146484375, 0.77490234375, 0.308349609375, -0.246337890625, -0.94775390625, 0.434814453125, 0.388916015625, -0.44189453125, -0.65576171875, -0.1368408203125, -0.53125, 0.09088134765625, -1.4501953125, 0.369384765625, -0.84326171875, 1.365234375, 0.5849609375, 0.11212158203125, -2.341796875, -0.487548828125, -0.450927734375, -1.5205078125, 1.6748046875, 0.08599853515625, -0.951171875, 1.4775390625, 1.798828125, -0.005123138427734375, 0.1351318359375, -0.309326171875, -1.5810546875, 0.37451171875, -0.374267578125, 1.2490234375, -0.75830078125, 0.701171875, -2.013671875, -0.603515625, 0.9658203125, 1.052734375, 2.099609375, -1.0517578125, -0.71142578125, 0.384521484375, 1.052734375, 1.1376953125, -0.72509765625, -1.0908203125, -0.2178955078125, -0.0217132568359375, -0.935546875, 0.295654296875, 1.09765625, -1.4541015625, -0.0256500244140625, 0.556640625, 1.134765625, -2.14453125, 1.40234375, 0.60888671875, 0.419921875, -0.10089111328125, -0.105224609375, 1.4091796875, 0.4013671875, 0.2958984375, 0.2059326171875, 0.49755859375, 0.251708984375, 0.70703125, 0.298095703125, 0.5634765625, 0.281982421875, 1.72265625, 0.421630859375, 0.10626220703125, -1.66796875, 0.1849365234375, 1.4267578125, 0.83642578125, 1.3203125, -0.227294921875, 1.1865234375, -0.76318359375, 0.484619140625, -0.658203125, 1.8701171875, -0.818359375, -1.6494140625, 0.240234375, -0.0765380859375, -1.7607421875, -0.8623046875, 0.97998046875, 0.053070068359375, 1.3349609375, 0.58056640625, 1.73046875, 0.216552734375, 0.5361328125, 1.6123046875, 1.470703125, 1.4443359375, 0.52197265625, -0.85205078125, 1.05859375, -0.006893157958984375, 0.2802734375, -1.0341796875, -1.0673828125, 0.4189453125, 0.80419921875, -1.2158203125, 0.240966796875, 0.51416015625, -0.063232421875, 1.4619140625, -1.0908203125, -0.9423828125, -0.055328369140625, -0.1676025390625, -1.0341796875, -0.62158203125, -1.029296875, -1.0751953125, -0.9521484375, 1.888671875, 1.35546875, 0.114013671875, -0.31884765625, -0.292236328125, 0.23779296875, -0.70849609375, -0.1192626953125, 1.1552734375, -1.7919921875, -0.153076171875, -0.8779296875, 0.4052734375, 0.8212890625, 0.650390625, 0.166748046875, 1.912109375, 0.08245849609375, -1.357421875, 0.5400390625, -0.73779296875, -0.1083984375, -0.5634765625, 0.87548828125, -0.52734375, 1.8173828125, -1.7802734375, -0.8642578125, 0.447509765625, -1.099609375, -1.5673828125, 0.9375, 0.419677734375, 1.138671875, -1.9345703125, 1.0439453125, -1.17578125, 0.73046875, 0.34765625, -1.4775390625, -0.430419921875, 0.05975341796875, -1.4248046875, -1.390625, 1.837890625, 0.1573486328125, 0.1612548828125, 0.1220703125, 0.37939453125, -0.2337646484375, -0.125732421875, -0.1380615234375, 0.2227783203125, -0.00406646728515625, 0.303466796875, -3.02734375, 0.58349609375, -0.87939453125, -0.43017578125, -0.453125, -0.40673828125, -0.66259765625, -0.1282958984375, 0.050872802734375, 0.6552734375, 0.87158203125, 1.228515625, 0.798828125, -0.18359375, -0.64697265625, 0.085205078125, -0.044219970703125, 0.006046295166015625, 0.9521484375, -0.037445068359375, 1.830078125, -1.05859375, -0.0247039794921875, 0.572265625, 0.837890625, 1.1533203125, -1.037109375, -0.36083984375, -1.5732421875, -1.6064453125, -0.8818359375, -0.12481689453125, -1.748046875, 0.427734375, 1.1904296875, -0.33740234375, -0.447998046875, -1.736328125, -1.4677734375, 1.806640625, -1.3291015625, -1.5458984375, -1.1494140625, 0.33056640625, -0.65966796875, 0.57666015625, 1.0693359375, 0.66357421875, -1.0283203125, -0.24267578125, -0.1007080078125, 2.447265625, 0.58642578125, -0.205322265625, -0.052001953125, -0.1292724609375, 1.0380859375, 0.880859375, -0.335205078125, 2.4140625, -0.61474609375, -0.81103515625, -0.316650390625, -0.783203125, 1.2529296875, 0.26171875, -0.4892578125, 1.6357421875, 0.6318359375, 0.0673828125, 0.83935546875, 1.0107421875, -0.78173828125, 0.449951171875, 1.2626953125, 2.8046875, 1.2568359375, -2.220703125, -1.6279296875, -0.5458984375, -0.258544921875, -1.076171875, 0.092529296875, 0.5146484375, 0.040435791015625, -0.35595703125, 1.3359375, -1.4091796875, -1.59765625, 1.087890625, 1.3603515625, -1.9052734375, -0.7822265625, 1.685546875, 0.161376953125, -0.82421875, -0.83544921875, 0.499267578125, 0.042755126953125, 0.283203125, 0.27734375, -2.416015625, -0.0283355712890625, -0.371826171875, 0.1407470703125, -0.036712646484375, 0.1763916015625, -0.947265625, -0.5546875, -0.65869140625, -1.7734375, -0.442626953125, 1.0400390625, 0.1058349609375, -0.441162109375, 1.052734375, -0.17431640625, -0.013214111328125, 0.517578125, 1.32421875, -0.7119140625, -0.248779296875, 0.373046875, 0.82861328125, 0.1639404296875, 0.234130859375, -0.225341796875, -1.17578125, 0.578125, -0.32666015625, 2.498046875, -0.12152099609375, -1.0263671875, -0.234130859375, 0.64111328125, 0.79150390625, -0.1417236328125, -0.90673828125, 0.10687255859375, 0.0083160400390625, -0.91162109375, -0.958984375, -1.1015625, -0.00384521484375, -1.3193359375, 0.388427734375, -0.256103515625, -1.21875, -0.7763671875, -0.385986328125, 2.1953125, 0.83837890625, 0.5888671875, -0.25439453125, -0.031890869140625, -1.7607421875, -0.03778076171875, -0.3955078125, -0.67041015625, 0.380859375, 0.47412109375, 0.43798828125, -1.47265625, 0.02325439453125, -1.490234375, 0.2064208984375, -0.004306793212890625, -0.420654296875, -0.0816650390625, -2.951171875, -0.35302734375, -0.08905029296875, 0.75732421875, -1.28515625, 2.3125, -1.125, 0.061065673828125, 1.5634765625, 0.5888671875, 1.32421875, 0.82958984375, 1.2333984375, -0.42626953125, 0.54541015625, -0.473388671875, 0.485107421875, 0.82275390625, -0.89501953125, 1.099609375, -1.251953125, 0.28564453125, 0.1556396484375, 0.9228515625, -1.013671875, -0.096923828125, -1.6728515625, 0.904296875, 1.15625, -0.221923828125, 0.66796875, -1.375, 0.0254669189453125, 0.2479248046875, 1.26171875, 0.80859375, 1.5126953125, -0.1033935546875, -0.403564453125, -1.0068359375, -0.75732421875, 1.509765625, 0.783203125, 2.814453125, -1.7734375, 0.40576171875, 0.77001953125, -0.155517578125, -0.3916015625, 0.2430419921875, 2.140625, 1.712890625, 0.060760498046875, 0.03179931640625, 1.5771484375, 0.464599609375, -0.36767578125, -1.4013671875, -0.181884765625, -0.040985107421875, 0.197998046875, 1.205078125, -0.06866455078125, -2.017578125, -1.81640625, -0.029083251953125, 0.7041015625, -0.1468505859375, -0.7763671875, -0.54638671875, 0.96826171875, -1.173828125, -1.5009765625, 0.1104736328125, -1.5302734375, -0.59716796875, -0.1802978515625, 0.1650390625, 0.30224609375, 0.449462890625, 0.72607421875, -0.10040283203125, -0.58251953125, -2.021484375, 0.1568603515625, 0.2110595703125, -0.2242431640625, -0.8642578125, 0.71142578125, 0.52392578125, 1.12109375, 0.30224609375, -0.145751953125, 0.28369140625, 1.0732421875, 0.85205078125, 1.0595703125, -0.25244140625, -0.58349609375, 0.60546875, -0.8359375, -0.669921875, -1.220703125, -0.609375, -0.447021484375, -0.08746337890625, 1.00390625, -1.26171875, 0.36865234375, 0.765625, -0.87109375, 0.33837890625, 1.9462890625, 0.11187744140625, 0.2958984375, 1.10546875, 1.53515625, -0.36279296875, -0.8408203125, 3.40234375, 0.77880859375, 1.0673828125, 0.58642578125, 1.244140625, -1.05078125, 0.53955078125, 1.3056640625, -1.767578125, 0.72119140625, -0.7939453125, -1.01171875, 0.32861328125, -2.3984375, 0.186767578125, -0.0948486328125, -0.479248046875, 0.318115234375, -0.9169921875, 0.053497314453125, 0.0546875, 1.4560546875, -0.22802734375, -0.66015625, 0.00811767578125, 2.029296875, -0.1695556640625, 0.06805419921875, -0.1878662109375, 0.032257080078125, 0.1175537109375, -1.85546875, 0.4287109375, 1.05078125, 0.06915283203125, 0.422607421875, 0.57177734375, 1.583984375, 0.60302734375, 0.008392333984375, -0.443603515625, -0.27587890625, 0.64892578125, 0.472412109375, -0.1788330078125, 0.415771484375, 0.83984375, -0.87353515625, -0.43212890625, -1.216796875, 0.26806640625, 1.16015625, 0.04443359375, 1.7939453125, 1.1787109375, 0.44384765625, -0.1602783203125, -0.1929931640625, -1.103515625, 1.2109375, -0.03631591796875, 1.470703125, -0.25, 0.5205078125, 0.89599609375, 0.48193359375, -0.8251953125, -0.53759765625, -0.051116943359375, -0.410888671875, -0.53125, -0.451904296875, 1.1484375, -1.4697265625, 1.6396484375, -0.2349853515625, -1.35546875, -0.529296875, 0.289794921875, 0.274658203125, -0.10498046875, -0.2841796875, 0.76123046875, 0.01042938232421875, -0.09228515625, -0.60107421875, 0.211181640625, 0.0731201171875, -0.740234375, -0.388671875, 0.1612548828125, 1.162109375, -0.9638671875, 0.1666259765625, 1.2890625, -0.54736328125, 0.369384765625, 0.85986328125, 0.1378173828125, 0.775390625, 0.2003173828125, -0.837890625, -0.391357421875, -0.1622314453125, -1.28515625, -1.107421875, -1.7568359375, -0.173095703125, -1.6767578125, 0.1065673828125, 0.99560546875, 1.49609375, 1.12109375, -0.1484375, -1.326171875, 0.857421875, -0.84326171875, 0.7060546875, 0.1444091796875, -1.55859375, 0.9990234375, -0.2371826171875, -1.697265625, -1.0, 1.7470703125, -0.6826171875, 0.161865234375, -0.255126953125, -0.73779296875, 0.0662841796875, -0.291748046875, 0.160888671875, 0.0570068359375, -0.3564453125, -1.9501953125, -0.53125, -1.5703125, 0.1546630859375, 1.3349609375, 1.9189453125, 0.10784912109375, 1.939453125, -0.07635498046875, 0.048248291015625, -1.701171875, 0.476806640625, -1.7568359375, -1.611328125, -1.55859375, -1.251953125, -0.8916015625, -0.2298583984375, 2.099609375, 0.411865234375, 0.68115234375, 1.86328125, -1.6044921875, 0.0011348724365234375, 0.93408203125, 1.513671875, -0.054718017578125, -1.458984375, -1.5947265625, 0.496826171875, 0.44140625, -0.65185546875, -0.76708984375, 0.693359375, -1.546875, -0.53369140625, 0.09912109375, -1.0166015625, 1.42578125, -0.8662109375, -1.1376953125, 1.326171875, 1.6328125, -0.41162109375, -1.0615234375, -2.05859375, -0.53564453125, -2.0, 0.094482421875, -0.64453125, 0.1893310546875, 0.25244140625, -0.85791015625, 0.0019292831420898438, 0.305419921875, -2.3046875, -1.6630859375, 0.22021484375, 0.422119140625, -0.63232421875, 0.3662109375, 0.36474609375, -0.48876953125, -0.419189453125, -1.416015625, 1.2109375, 0.99365234375, -0.2225341796875, -1.919921875, -0.990234375, -1.5146484375, 0.44140625, -1.21484375, -0.73828125, -0.1636962890625, -1.5595703125, -0.050140380859375, -0.5771484375, 0.775390625, -0.73828125, -0.62841796875, -0.5390625, 0.1529541015625, -0.488037109375, 0.71875, -1.4404296875, -0.58056640625, -0.137939453125, 1.6884765625, 0.358154296875, 0.304443359375, -0.16455078125, -0.484130859375, 0.58251953125, -0.583984375, -1.2763671875, 0.76025390625, -0.763671875, -0.662109375, -0.44921875, -0.1502685546875, -0.321044921875, -0.623046875, -0.78271484375, 0.56689453125, -1.771484375, 0.1302490234375, -0.5859375, -0.324462890625, -0.5478515625, 1.498046875, 0.1661376953125, -1.4736328125, -1.298828125, 1.658203125, 1.25390625, -0.78466796875, 0.345703125, -0.1385498046875, -0.91796875, -0.677734375, -0.724609375, -0.453857421875, 0.9462890625, -1.03515625, 0.0247344970703125, -1.029296875, -1.7041015625, 0.96435546875, -1.5537109375, 0.6298828125, -0.99267578125, 0.0526123046875, -0.8837890625, -0.5478515625, -0.439697265625, -0.429443359375, -1.2734375, 1.2861328125, 0.42626953125, 0.08416748046875, -1.02734375, 0.0411376953125, 0.63818359375, -0.5224609375, -0.7080078125, 0.2568359375, -0.9033203125, -1.5869140625, -1.2919921875, -0.84228515625, 1.1806640625, 0.07843017578125, -2.005859375, 1.7041015625, -0.60107421875, -0.7568359375, -0.66552734375, -1.4931640625, -0.092529296875, -0.3291015625, -0.44140625, 0.97412109375, 0.71923828125, -0.067138671875, 0.95556640625, -1.0068359375, 0.443603515625, 0.18310546875, 1.125, 0.4111328125, 0.73876953125, -0.8837890625, -0.24365234375, -0.34619140625, 0.57861328125, -1.16796875, -0.814453125, -2.23046875, -0.521484375, 2.140625, 0.49169921875, -0.9443359375, -2.083984375, -0.576171875, -0.01776123046875, -0.95263671875, 0.26953125, 0.1331787109375, 0.6884765625, -0.8125, -0.087646484375, 1.1591796875, 1.5166015625, 0.94775390625, 1.2392578125, -1.3583984375, 0.30322265625, 0.10260009765625, -1.494140625, -0.1507568359375, 0.208740234375, -0.78515625, 0.08984375, -0.03369140625, 0.90283203125, 0.2474365234375, -0.93310546875, 0.08209228515625, 2.259765625, -1.40625, -1.48828125, -0.5, 0.455810546875, -1.615234375, -0.2064208984375, -0.640625, -0.732421875, 1.333984375, 1.31640625, -0.56689453125, 0.84130859375, 0.442138671875, -0.2021484375, -2.20703125, 0.64892578125, -1.1005859375, 1.8671875, -2.0234375, 0.55908203125, 0.1826171875, -0.3232421875, -2.8515625, 1.1708984375, 0.059906005859375, -2.890625, 1.328125, -2.2890625, -0.51025390625, 1.16015625, 0.3046875, -0.93798828125, -0.56201171875, -0.99658203125, -0.354248046875, 0.8798828125, 0.58203125, -0.400390625, 0.86669921875, -1.5107421875, 0.40283203125, -0.1766357421875, -0.2763671875, -1.576171875, 1.3369140625, 1.3828125, -1.2353515625, 1.626953125, -1.3564453125, -0.279052734375, 1.3291015625, 0.057586669921875, -0.353271484375, 0.09344482421875, -0.2171630859375, -0.491455078125, -0.08624267578125, 0.3701171875, 0.72021484375, 0.464599609375, -0.650390625, -0.12310791015625, 0.6123046875, -0.92529296875, 0.69580078125, 0.1910400390625, -0.032989501953125, -0.60009765625, 0.5771484375, -1.572265625, 0.86083984375, 0.328125, 0.335693359375, -0.364990234375, -0.71923828125, -0.36328125, -1.1474609375, -0.08111572265625, -0.43798828125, 1.2109375, -2.25390625, 0.424560546875, 0.77880859375, 0.351318359375, -0.43798828125, -2.03515625, -0.226806640625, -0.80712890625, -1.802734375, 0.1392822265625, 0.47314453125, -0.71240234375, -0.39501953125, 1.3779296875, -0.0086212158203125, 0.5302734375, -0.10125732421875, 0.31982421875, 0.294921875, -0.396240234375, 0.335205078125, 0.140869140625, 1.05859375, 0.35791015625, 1.4619140625, 0.271484375, 0.443603515625, -0.044586181640625, 0.10662841796875, 0.693359375, 1.220703125, 0.0697021484375, 0.458740234375, -1.8125, -0.423828125, 0.703125, 1.5322265625, 0.43310546875, 0.8935546875, -0.330322265625, 0.302978515625, 1.662109375, -0.1280517578125, 0.2296142578125, 1.3876953125, -0.7294921875, -0.249755859375, 0.88427734375, 2.05859375, -1.1083984375, 0.027130126953125, -0.54248046875, 0.50732421875, -1.984375, 1.0146484375, -0.65869140625, 0.62353515625, 1.466796875, -0.0411376953125, 1.2421875, 1.6064453125, 0.814453125, 2.25390625, -0.443359375, -0.1680908203125, 0.630859375, -0.276611328125, -0.77734375, 0.8564453125, -1.2314453125, -0.2274169921875, -0.919921875, -0.833984375, 0.494873046875, 0.415283203125, 0.400390625, -0.0259552001953125, -0.318603515625, 0.403564453125, 0.55419921875, 0.05792236328125, 0.56201171875, 0.1624755859375, 0.416259765625, -1.037109375, 0.10064697265625, 0.9921875, 0.60693359375, 1.0283203125, -1.7275390625, -2.302734375, 0.13671875, 0.273681640625, -1.6845703125, 1.2041015625, 0.5966796875, 0.76904296875, 0.66259765625, -0.63671875, -1.2880859375, 0.385986328125, -0.6259765625, 0.2088623046875, 0.28173828125, 0.52490234375, -0.5166015625, 1.634765625, -0.7958984375, -1.0791015625, 0.8662109375, 0.257568359375, -0.96044921875, -0.9130859375, -0.324462890625, -0.50830078125, 1.5732421875, 0.56298828125, -1.69921875, -0.82666015625, -1.62109375, -1.01171875, 1.5390625, 0.2197265625, 0.1822509765625, -1.66015625, 0.1953125, 0.031768798828125, 0.82275390625, -0.61669921875, -2.626953125, 0.5908203125, 0.94287109375, -0.6455078125, -1.111328125, -1.1845703125, -0.4208984375, 0.5908203125, -0.261474609375, 0.7431640625, -0.407470703125, -1.5341796875, 0.491943359375, 2.3828125, -0.8525390625, -0.476806640625, -0.04095458984375, -0.18896484375, 0.712890625, 0.024566650390625, -0.98486328125, 0.2291259765625, -0.404052734375, 0.7802734375, -0.3369140625, -0.261474609375, 0.86865234375, -0.57763671875, -0.2578125, 0.03143310546875, -0.845703125, -2.005859375, -0.19287109375, 0.053863525390625, 0.224853515625, -0.5556640625, -1.693359375, -0.7041015625, -0.36865234375, 1.580078125, 0.70703125, 1.388671875, 0.293701171875, 0.34228515625, 1.521484375, -0.406982421875, -1.04296875, 0.96337890625, 0.12548828125, 1.83984375, 2.22265625, 0.35107421875, 0.306396484375, -1.0234375, 0.82470703125, 0.35546875, 0.8349609375, 0.43896484375, -1.0, 0.93115234375, 0.9248046875, -0.1331787109375, 0.52490234375, 0.71728515625, -0.230224609375, 0.740234375, -0.2432861328125, 0.97412109375, -0.4853515625, 1.3701171875, -0.87890625, 0.08843994140625, 0.10650634765625, -1.2958984375, 0.88330078125, 0.7734375, 0.8193359375, -0.298583984375, 0.465576171875, -0.77197265625, 1.28515625, 1.412109375, -0.35595703125, 0.98681640625, 1.1142578125, 0.10504150390625, 0.128173828125, 1.595703125, -0.38134765625, -0.9248046875, 0.8349609375, 0.0899658203125, -0.424072265625, 0.174072265625, 1.310546875, -1.216796875, 0.51123046875, 0.52587890625, 0.244384765625, 2.279296875, -0.1641845703125, -0.484375, -0.7490234375, 1.259765625, 2.8359375, -1.4970703125, -0.5185546875, 0.38134765625, -0.5361328125, -0.01494598388671875, 2.728515625, 0.330322265625, 0.64404296875, 1.171875, -0.026702880859375, 0.322021484375, -1.4931640625, 0.7578125, -0.498779296875, -0.383544921875, 1.3505859375, -0.430419921875, 0.45361328125, 0.06719970703125, 0.062042236328125, 0.32080078125, -0.5107421875, -1.6728515625, -0.99462890625, -1.697265625, 0.10894775390625, -2.205078125, -1.083984375, 0.055755615234375, 0.31982421875, 0.55224609375, 1.4541015625, -1.53125, 2.26171875, 0.3525390625, -1.6572265625, -0.0277862548828125, -0.9697265625, 1.783203125, 0.1422119140625, -0.328369140625, 1.072265625, 0.047607421875, 0.3525390625, 0.2418212890625, 0.94677734375, 0.85205078125, 0.1923828125, -0.72314453125, 0.439697265625, 0.1221923828125, -0.4521484375, 0.54443359375, 0.2081298828125, 1.9560546875, 0.357666015625, 2.15625, 1.02734375, -1.16015625, 1.0166015625, -0.201416015625, 0.0020294189453125, 0.0843505859375, 0.9501953125, -1.0283203125, 0.397216796875, 1.34375, -2.123046875, -1.28125, -0.0096893310546875, 0.397705078125, -1.9501953125, 0.498779296875, -0.58544921875, 1.2001953125, -0.409912109375, 0.444091796875, 0.3525390625, 0.1474609375, 0.48095703125, 1.0654296875, 1.0, -1.451171875, 1.7353515625, 0.409912109375, -1.3369140625, 0.8125, 0.438232421875, -0.01251983642578125, 0.94873046875, -0.252685546875, 0.0609130859375, 1.34375, 0.87255859375, 1.3779296875, -0.3798828125, 0.492431640625, 0.67626953125, 0.13720703125, 1.314453125, -0.34375, 1.1484375, -1.7392578125, -0.64111328125, -0.219970703125, 1.294921875, -0.55224609375, 0.62744140625, 1.66796875, 1.52734375, 0.9716796875, -0.76318359375, 0.2158203125, 0.48193359375, 0.6806640625, -0.8828125, 1.4609375, 0.84130859375, -0.51953125, 1.3603515625, 0.44482421875, -1.90625, 2.18359375, 0.264404296875, -1.4375, 0.7578125, -1.4501953125, 0.282470703125, 0.31884765625, 1.380859375, 0.638671875, -0.32763671875, -1.3818359375, -1.2548828125, 0.386962890625, -2.05859375, 0.830078125, -0.347412109375, 0.315673828125, 2.275390625, -0.60009765625, 0.39453125, 1.14453125, -1.2626953125, 0.017974853515625, 1.04296875, -0.5849609375, 0.783203125, -0.07061767578125, 0.270263671875, -0.8388671875, -2.564453125, 0.6533203125, 0.216064453125, 0.9052734375, 0.5087890625, -1.6171875, 0.348876953125, -0.444091796875, -0.288330078125, -0.29345703125, 0.3447265625, -0.485595703125, 1.0615234375, 0.491943359375, 1.482421875, -0.46728515625, 0.87548828125, -2.220703125, 0.378173828125, -0.5908203125, -0.431640625, -0.2333984375, -1.115234375, 0.60595703125, -0.09063720703125, -0.802734375, 1.134765625, 0.8193359375, -0.515625, -0.2431640625, 0.74462890625, -1.1650390625, 0.439453125, 0.0201873779296875, 1.1484375, -0.6357421875, -1.0478515625, -0.150146484375, 1.0751953125, 0.68017578125, 1.5478515625, -0.609375, 0.469482421875, -0.54443359375, 0.059356689453125, 0.11712646484375, -1.111328125, -0.1990966796875, -0.35302734375, -1.275390625, 1.39453125, 1.138671875, -0.322998046875, 0.413818359375, 0.57470703125, -0.6884765625, 1.8642578125, 0.95361328125, 0.82275390625, 1.7451171875, -0.88330078125, 0.51318359375, -0.383544921875, -1.046875, -0.056365966796875, 0.7763671875, -0.311767578125, 0.77490234375, -0.91748046875, -0.591796875, 2.203125, 0.90869140625, 0.013153076171875, -0.431640625, 0.7763671875, -1.583984375, -0.06842041015625, 0.060546875, 1.0390625, 2.6015625, -0.1241455078125, -1.322265625, 1.263671875, 0.28857421875, -1.9150390625, -0.472412109375, 1.7431640625, -0.485107421875, -0.11260986328125, -1.7568359375, -0.79736328125, 0.081298828125, 0.57177734375, 0.419921875, -0.40478515625, 0.0007047653198242188, 1.5849609375, 1.55859375, 0.061981201171875, 0.7158203125, 0.239013671875, -1.2060546875, -0.155517578125, 1.8701171875, -0.740234375, -1.2119140625, 0.96435546875, -0.52880859375, 0.2174072265625, -0.056243896484375, -1.26171875, -1.2763671875, 0.98291015625, -0.10968017578125, -2.998046875, -1.7451171875, -0.60009765625, 0.003421783447265625, -0.6875, 1.5107421875, -0.59423828125, -0.48486328125, 0.0645751953125, 0.91748046875, -0.85205078125, -1.8505859375, 1.55859375, -1.7900390625, 0.91357421875, 0.2349853515625, 0.74658203125, -0.5908203125, 0.361572265625, 0.30078125, 0.82177734375, 0.47509765625, 0.86474609375, -0.4189453125, -0.54931640625, 0.372314453125, -0.11407470703125, 1.7822265625, 0.94775390625, -0.58447265625, -1.021484375, -0.104248046875, -0.94384765625, -0.314697265625, -0.393798828125, -0.9248046875, 0.3154296875, 1.4130859375, 0.1536865234375, -0.87939453125, -0.339599609375, 0.60693359375, -0.05474853515625, 0.5078125, -0.046600341796875, 0.55712890625, -1.0302734375, 0.1446533203125, -1.5947265625, 1.1220703125, 0.56884765625, -0.261474609375, -1.3251953125, -0.301025390625, 1.9228515625, -0.3515625, -0.1234130859375, 1.42578125, 0.57568359375, 1.998046875, -0.80517578125, -0.052886962890625, -1.21484375, -2.32421875, 0.96533203125, 0.6640625, 2.21484375, 0.06915283203125, 0.365234375, -0.2169189453125, -0.6962890625, 0.2027587890625, 0.5634765625, 0.7333984375, -0.998046875, -0.3740234375, -0.390625, 0.85009765625, 1.1962890625, 0.453857421875, -0.8046875, 0.09051513671875, 1.4658203125, -0.62451171875, -0.98974609375, -0.50830078125, -0.053436279296875, 0.42724609375, -0.6533203125, 0.61181640625, -1.1474609375, 0.95751953125, 0.76611328125, 1.2939453125, -0.58837890625, -0.70556640625, -0.1318359375, 1.5703125, -1.8642578125, 0.477783203125, 0.95947265625, -1.076171875, 1.162109375, 0.41748046875, -0.40771484375, 0.65087890625, 0.03765869140625, 0.60302734375, -0.460205078125, -1.9951171875, 1.15625, -0.11260986328125, -1.3203125, 0.296875, -0.740234375, -0.37646484375, 1.38671875, 0.1138916015625, -0.517578125, 1.1142578125, 0.75439453125, 0.21142578125, 0.01418304443359375, 0.03973388671875, 0.043426513671875, 1.21875, 0.04705810546875, 1.650390625, -0.359130859375, -1.353515625, 1.40625, 0.236572265625, 0.97021484375, 0.3984375, -0.75390625, 0.43115234375, 1.90234375, -2.126953125, 1.4580078125, -1.3232421875, 1.6689453125, 0.91259765625, -0.7783203125, 0.400634765625, 0.83251953125, -0.6669921875, 0.31201171875, -1.35546875, -2.443359375, -1.5234375, -0.84130859375, -1.13671875, -2.017578125, -0.00724029541015625, 0.196533203125, -1.8798828125, -0.35986328125, -1.0751953125, -2.7265625, 0.9287109375, -2.37109375, -1.5693359375, 0.35791015625, -0.31689453125, 0.1290283203125, -1.2646484375, -1.2470703125, -1.1748046875, -1.751953125, 1.2060546875, -0.720703125, 0.908203125, -0.7490234375, 0.34619140625, -0.1307373046875, 0.42041015625, -0.65576171875, 0.82958984375, -0.51708984375, -0.423583984375, 0.49072265625, 0.845703125, -0.46533203125, 0.33740234375, -0.003314971923828125, 0.84619140625, 0.00797271728515625, -1.54296875, 0.435791015625, 1.2431640625, 0.232666015625, 0.2447509765625, 0.91259765625, 0.66357421875, 1.21875, -1.5537109375, 0.44091796875, 0.037628173828125, 0.2156982421875, 0.85302734375, -0.5673828125, 1.416015625, -1.4091796875, -1.6728515625, -0.78125, 0.99267578125, -0.04510498046875, 0.75048828125, -0.1474609375, 0.41943359375, 0.11334228515625, 1.28515625, 1.7890625, -0.75732421875, 0.2176513671875, 0.71435546875, -0.62646484375, 0.08062744140625, -1.2705078125, -0.369140625, 0.357666015625, 0.5439453125, -0.1612548828125, -0.9794921875, -0.0307159423828125, 1.1611328125, 0.5322265625, -1.509765625, -1.40234375, 1.396484375, 0.2255859375, -0.85302734375, -0.892578125, 0.0797119140625, 0.264892578125, -1.8193359375, -0.408447265625, 0.1099853515625, -0.493408203125, 0.67724609375, 0.397705078125, -0.295654296875, -1.708984375, 0.4873046875, 1.1728515625, 1.078125, 0.595703125, -0.456787109375, 2.34765625, -0.81591796875, 0.05657958984375, -1.00390625, 0.54248046875, 0.11151123046875, 0.1263427734375, -1.34765625, -0.190185546875, 2.974609375, -0.173828125, 0.489990234375, -2.724609375, -1.25, -0.6904296875, -0.475830078125, 0.98974609375, -0.9482421875, -0.461181640625, 0.67724609375, -1.9892578125, 0.1373291015625, -0.67529296875, -1.1484375, -1.53125, -0.44921875, -1.9111328125, -0.3671875, -0.16748046875, 1.0546875, -0.14990234375, 1.5185546875, -0.13525390625, -0.66650390625, -0.07171630859375, -0.69140625, -1.806640625, -0.31689453125, -2.34375, 0.53466796875, -0.07452392578125, -0.095703125, -0.78369140625, 0.444580078125, -0.5146484375, -0.1937255859375, -1.26171875, 1.1416015625, 2.03125, 0.132568359375, 0.72021484375, -0.369140625, -0.8515625, 0.80908203125, 0.331787109375, 1.3994140625, 0.07275390625, -0.82568359375, 2.232421875, 0.8515625, 0.31201171875, -0.521484375, -1.205078125, 2.681640625, 0.09478759765625, -1.3955078125, 0.03076171875, 0.91259765625, 0.114501953125, 0.70166015625, -1.078125, -1.416015625, -0.01311492919921875, -0.37548828125, -1.0400390625, -1.27734375, -0.298828125, 0.87548828125, 1.654296875, -0.1536865234375, 1.498046875, 0.77197265625, 0.73974609375, -0.0186309814453125, -0.7666015625, 0.14794921875, -1.283203125, -2.3671875, 0.6552734375, -1.154296875, 0.4951171875, -0.31591796875, -2.0546875, 0.11444091796875, -1.8447265625, -1.3310546875, 1.1142578125, 0.30810546875, -0.439208984375, -1.427734375, 0.53076171875, 1.1875, 0.43115234375, 0.62353515625, -0.58154296875, -0.72412109375, -0.2113037109375, -1.0654296875, -0.423095703125, -0.38916015625, -0.217041015625, 0.71630859375, 0.477294921875, -1.197265625, 1.9013671875, 0.76708984375, 0.53076171875, 0.65087890625, -3.087890625, -0.607421875, 0.64599609375, -0.1995849609375, 1.9580078125, -1.033203125, -1.404296875, -0.0660400390625, -0.0181884765625, -0.5048828125, 1.3076171875, 0.0140533447265625, 0.053253173828125, 0.74951171875, 0.1817626953125, 1.232421875, -1.7529296875, 0.4892578125, -1.0078125, 1.2587890625, 1.0693359375, -0.2254638671875, -1.0966796875, 0.8115234375, 1.9482421875, 1.0439453125, 0.0400390625, 0.235595703125, -0.52783203125, -0.6611328125, 1.87890625, 0.1776123046875, 0.4169921875, 0.8818359375, -0.490234375, 0.30126953125, -0.53173828125, 0.79638671875, 0.140869140625, 1.4775390625, -0.5185546875, 1.98046875, 0.1268310546875, -2.1015625, 0.356201171875, 0.1759033203125, -0.31689453125, 0.966796875, 0.350341796875, 0.80224609375, -0.67626953125, -0.9609375, 1.646484375, -1.4775390625, -0.94384765625, -0.7587890625, -0.54443359375, 1.078125, 0.304931640625, -1.4140625, -0.88037109375, 1.3876953125, 0.251220703125, -0.463623046875, -0.462158203125, 1.1865234375, -0.76220703125, 0.50439453125, -0.83056640625, -0.5419921875, -0.060882568359375, 1.2900390625, 1.544921875, 0.62158203125, -0.78564453125, -2.26171875, -1.31640625, 1.5537109375, 0.59423828125, 0.8359375, -1.8466796875, 0.0714111328125, 0.76953125, -1.23828125, 0.407958984375, 0.317626953125, -0.31298828125, 0.751953125, 0.60302734375, -0.76953125, -1.2763671875, 0.4365234375, 0.318359375, 0.389404296875, 0.468017578125, 0.90625, 1.8173828125, 0.42431640625, 1.8349609375, -0.30029296875, 0.90087890625, 0.474365234375, -0.732421875, -0.28564453125, 0.2041015625, -0.79833984375, -0.91064453125, 0.6357421875, 0.5263671875, -1.2392578125, 1.0341796875, -0.15185546875, -1.267578125, -0.61083984375, -0.003086090087890625, 0.09747314453125, -0.402587890625, 0.250244140625, 0.6708984375, -1.060546875, -0.0269317626953125, 1.146484375, -0.2159423828125, 0.7080078125, -0.20263671875, -0.89892578125, 0.306396484375, -0.4501953125, 0.420166015625, 1.87109375, 0.63037109375, 0.213134765625, 0.30517578125, -0.5576171875, 0.82177734375, 0.45166015625, -0.04742431640625, -1.4287109375, -0.5703125, 0.7353515625, 0.0589599609375, 1.294921875, 0.830078125, -0.83740234375, 0.71923828125, -1.47265625, -0.485107421875, 0.9765625, -0.380615234375, 0.5625, 0.04669189453125, -0.08111572265625, 0.611328125, 0.450927734375, -0.449951171875, -0.344970703125, 0.06146240234375, -1.3046875, 0.72216796875, 1.0048828125, -0.7900390625, -0.1566162109375, 1.44921875, -1.029296875, -0.465087890625, 0.257568359375, 1.111328125, 0.380615234375, 0.81884765625, -1.5341796875, 1.501953125, 0.055450439453125, 0.201171875, 0.0401611328125, 1.119140625, 1.0732421875, -1.1396484375, -0.088134765625, 0.61572265625, -0.084716796875, -0.130126953125, 0.144775390625, 3.03125, 0.1793212890625, -0.3837890625, -0.16796875, -1.0966796875, -0.419189453125, -0.1422119140625, 1.771484375, 2.158203125, -1.419921875, -0.53369140625, 1.5693359375, 0.215576171875, -0.2447509765625, 0.04119873046875, -1.1982421875, -1.1064453125, -0.86962890625, -0.25634765625, 0.063232421875, 0.50341796875, 0.414306640625, -0.45654296875, 2.11328125, -1.689453125, 1.6005859375, -0.7236328125, -0.439453125, -0.568359375, 0.5546875, -2.087890625, 0.55078125, -0.5634765625, 0.319091796875, 0.7021484375, -0.75927734375, -0.65234375, 0.051605224609375, 0.83642578125, 2.41015625, -1.95703125, 1.0517578125, -0.94140625, 1.39453125, -0.8505859375, -0.74755859375, -0.281494140625, -0.56201171875, -0.409912109375, 0.9345703125, 1.4501953125, -1.373046875, 0.5810546875, 1.9404296875, -1.27734375, -0.326416015625, 0.430908203125, 0.72119140625, 0.4755859375, 0.68701171875, -0.60546875, 0.275146484375, 1.4951171875, -0.12060546875, -0.183837890625, 0.96044921875, -0.372802734375, 1.39453125, -0.99658203125, 0.5771484375, 0.9921875, -1.5986328125, 0.81396484375, 1.23828125, -0.15576171875, 0.96044921875, -0.9873046875, -0.004352569580078125, 0.515625, -0.379638671875, -1.6533203125, 0.80419921875, 0.239013671875, -0.61083984375, -0.476318359375, -0.02471923828125, 0.57666015625, -0.55712890625, -0.340087890625, 0.1370849609375, -0.59765625, 0.176025390625, 0.252197265625, -1.443359375, -0.8125, -1.5498046875, -0.269287109375, -1.6787109375, -0.6728515625, 0.79150390625, 0.455322265625, -1.87109375, 0.1456298828125, 0.68212890625, -1.0390625, 0.78515625, 0.4716796875, -1.10546875, -2.904296875, -0.83740234375, -0.090087890625, 1.12109375, -1.0400390625, -1.5712890625, -1.822265625, -1.8232421875, 0.927734375, 0.47509765625, -0.058013916015625, -0.099609375, -0.0882568359375, -1.4599609375, -1.6181640625, -0.6767578125, 1.7470703125, -0.5283203125, -0.74951171875, -1.212890625, -0.75146484375, -0.2100830078125, -1.6337890625, 0.888671875, -0.69140625, 1.7998046875, -0.60546875, 0.20361328125, 0.654296875, -0.2685546875, -0.28271484375, -0.39404296875, -0.5048828125, -0.206298828125, -1.7578125, 1.26953125, 1.5166015625, -0.0192108154296875, 1.798828125, 0.393798828125, 0.311279296875, 0.380615234375, -0.73095703125, -1.4306640625, -0.26220703125, 0.5029296875, -0.27880859375, 1.0791015625, 0.3515625, -0.65576171875, 0.6884765625, -1.5625, 0.462158203125, 0.302490234375, 1.1513671875, 1.623046875, -0.81640625, -1.48828125, -0.1517333984375, 0.7001953125, 1.4638671875, 0.66162109375, -0.43798828125, 0.256591796875, -0.85107421875, 0.91943359375, -1.1865234375, 1.2724609375, 1.38671875, 0.1728515625, 1.9560546875, -1.7978515625, -2.34765625, -1.568359375, -0.374267578125, 0.1607666015625, 0.3154296875, 0.787109375, -2.671875, -0.8974609375, -1.03125, 1.0732421875, -0.32080078125, -0.03955078125, 1.0341796875, -0.11181640625, -1.57421875, -0.861328125, -1.4931640625, -0.285400390625, -2.111328125, -0.474609375, -0.032867431640625, 1.138671875, 0.273681640625, -0.0225067138671875, 1.091796875, 0.8232421875, 0.77001953125, -0.85400390625, 0.01580810546875, -0.0589599609375, 2.02734375, -0.9873046875, -0.09393310546875, 1.0390625, -0.71923828125, -1.8798828125, -1.513671875, 1.3916015625, 0.94775390625, 0.347412109375, 0.36669921875, 0.93896484375, 0.315673828125, -0.9169921875, 1.33203125, 1.091796875, 0.0673828125, -0.544921875, -1.1298828125, -0.410400390625, 1.083984375, 0.26318359375, 0.2998046875, 1.9990234375, -1.9599609375, -0.93798828125, -1.4658203125, -0.6640625, -1.515625, 0.41162109375, -0.56103515625, -0.78173828125, 1.1416015625, -0.134033203125, 0.1002197265625, 0.133544921875, 0.7021484375, 0.475341796875, 0.517578125, -0.448486328125, -0.212646484375, 1.0341796875, -1.859375, 0.471435546875, -0.76318359375, 1.6044921875, 1.5478515625, 0.143310546875, -0.10992431640625, 0.6298828125, 0.708984375, 0.412353515625, -0.98388671875, -0.308837890625, 0.0758056640625, -0.2435302734375, -0.82666015625, 0.399658203125, 0.10174560546875, 0.42919921875, 0.023468017578125, -0.37890625, 1.884765625, -0.8935546875, 1.9296875, -1.205078125, -0.81201171875, -0.9609375, -0.5361328125, 0.040740966796875, -0.038665771484375, -0.80224609375, 0.3525390625, -0.32958984375, 0.02825927734375, 1.923828125, 0.96337890625, 0.87646484375, 0.5185546875, 0.1190185546875, 0.560546875, -0.48779296875, 0.07720947265625, 1.1279296875, -1.0146484375, -1.9853515625, 1.5009765625, 0.5302734375, 0.5986328125, -0.05810546875, -0.483154296875, -0.51220703125, 0.48388671875, -0.4189453125, -1.125, 0.07086181640625, -0.9521484375, -0.56591796875, 0.353759765625, 2.421875, -0.0709228515625, 0.7841796875, -1.705078125, 0.44287109375, 0.360107421875, 0.248291015625, -0.6171875, 0.42236328125, -1.05078125, -1.41796875, 0.468994140625, 1.689453125, 0.4921875, 0.9638671875, -0.378173828125, 0.434814453125, 0.075439453125, 0.1558837890625, 1.435546875, -0.2578125, 0.55810546875, 0.44580078125, -1.0576171875, -2.443359375, 0.1181640625, 1.609375, -0.52099609375, 0.237548828125, 0.66455078125, 0.3310546875, -0.485107421875, -0.96826171875, 1.427734375, -0.4658203125, -1.6123046875, -0.7373046875, -1.23046875, -0.3544921875, -0.26904296875, -0.11065673828125, 2.283203125, 2.3046875, -0.5234375, -1.7353515625, 0.77197265625, 2.068359375, -0.59716796875, 2.9921875, -2.09765625, -0.3203125, 0.81982421875, 0.65380859375, 0.07891845703125, -1.322265625, -1.017578125, 0.73974609375, -0.40283203125, -0.56005859375, 0.630859375, -0.96142578125, -0.78466796875, 0.5849609375, -0.150634765625, 1.6884765625, -0.77783203125, 0.51123046875, -1.4208984375, 0.2374267578125, 0.0872802734375, -0.054290771484375, 0.02362060546875, -1.4248046875, -0.77001953125, 0.010009765625, 1.3271484375, 0.301025390625, 0.4736328125, 0.99609375, -0.050445556640625, -0.2919921875, 0.96533203125, 0.137451171875, 1.46875, -0.495361328125, -0.8115234375, -0.421875, -0.93017578125, 1.8095703125, 0.6083984375, -0.3369140625, 0.306884765625, 0.82177734375, 0.437255859375, 0.390869140625, 1.0595703125, -1.537109375, -1.349609375, -1.4521484375, 0.10198974609375, 0.321044921875, -1.673828125, -1.2265625, 2.919921875, 1.775390625, 0.140380859375, 0.048370361328125, -1.220703125, -0.58740234375, -1.705078125, -0.448486328125, 0.10223388671875, 0.4931640625, 1.58203125, 0.876953125, -0.5400390625, -0.75048828125, -1.96875, -0.01276397705078125, 0.07708740234375, -0.00125885009765625, 2.044921875, 0.52880859375, 0.23681640625, -0.25732421875, -0.779296875, 0.546875, -0.80615234375, 0.94775390625, -0.233154296875, 0.11248779296875, -0.47607421875, 1.0556640625, 0.45361328125, -0.3154296875, 0.347900390625, -0.78515625, 0.396728515625, 1.369140625, -1.0625, -0.8251953125, -0.428955078125, -2.47265625, 1.0751953125, -0.0609130859375, 0.4052734375, 1.1494140625, 2.216796875, 0.0169677734375, 0.77978515625, -0.74365234375, -0.273193359375, 0.491943359375, 1.0224609375, 0.6103515625, -1.9833984375, 2.2578125, 0.82763671875, 1.123046875, -2.185546875, 1.1025390625, 0.60498046875, -0.744140625, 2.693359375, 1.0341796875, -0.1075439453125, -0.351318359375, 0.385498046875, -2.11328125, 0.6220703125, -1.0556640625, 0.52099609375, -0.89013671875, -1.623046875, -0.1590576171875, -0.058624267578125, 1.392578125, 2.72265625, 0.1807861328125, 0.1383056640625, 0.53857421875, -0.47509765625, -0.1817626953125, -0.46484375, 0.5537109375, 2.087890625, -1.3095703125, -0.1829833984375, -0.74072265625, -0.62451171875, 0.58447265625, 1.3203125, -0.055389404296875, 0.46142578125, 0.4404296875, -1.0400390625, -0.146728515625, -0.814453125, 0.1060791015625, 0.44189453125, -0.92529296875, -2.205078125, 0.654296875, -0.33935546875, 0.10614013671875, -0.2462158203125, -1.0869140625, -1.2861328125, -0.5732421875, 1.0439453125, -0.98486328125, 1.81640625, 0.4609375, -0.88671875, -1.5595703125, -1.3779296875, 0.168212890625, -0.12481689453125, 1.3837890625, -1.0986328125, -0.3603515625, -0.076416015625, 0.06402587890625, 0.045196533203125, 0.433349609375, -0.3623046875, 1.103515625, 0.427978515625, 2.755859375, 0.07501220703125, -0.350830078125, 0.1771240234375, 1.845703125, -1.6591796875, 0.376220703125, 1.126953125, -0.3154296875, -0.08984375, 0.90771484375, 0.329345703125, -0.037017822265625, -0.2705078125, -0.290771484375, -1.1435546875, -0.493408203125, -0.88232421875, 0.46728515625, -0.95361328125, -1.7587890625, 0.42236328125, 0.72509765625, 0.38818359375, -0.1256103515625, -2.0546875, -0.7080078125, 0.8544921875, 0.771484375, -0.90283203125, -1.4208984375, -0.1785888671875, -1.1103515625, 0.272705078125, 0.37548828125, -0.5400390625, 1.4765625, -0.56689453125, -1.1591796875, -2.5, -1.30078125, -0.9521484375, -0.275390625, 0.417724609375, 0.367919921875, -0.75390625, 0.9775390625, -0.2178955078125, 0.94970703125, -1.0517578125, -1.900390625, -0.67333984375, -0.9130859375, 2.21484375, -0.314453125, -1.296875, -0.892578125, 0.701171875, -0.923828125, 0.0970458984375, 1.2373046875, -1.701171875, -0.638671875, 1.5302734375, -0.28076171875, -0.404541015625, 0.12249755859375, -1.1044921875, -0.70703125, -0.2119140625, 0.88720703125, -0.427978515625, 0.068359375, -1.1044921875, -0.76953125, 0.0670166015625, 0.326416015625, 0.11444091796875, -0.9375, 0.053436279296875, -1.0322265625, -1.4306640625, -0.1685791015625, -0.151611328125, 0.294921875, -0.9482421875, 0.64990234375, 0.1903076171875, 0.1846923828125, 0.5625, 1.21875, -0.1641845703125, 0.783203125, -1.419921875, -1.6923828125, -0.46875, 1.1943359375, 1.76953125, -0.318115234375, -0.035247802734375, 1.12109375, 0.69384765625, 1.083984375, 1.677734375, 0.1712646484375, -0.56982421875, -1.025390625, -0.75146484375, -0.2298583984375, 0.45703125, 0.249755859375, 0.06939697265625, 1.291015625, 0.23388671875, -0.8916015625, 2.24609375, -1.3369140625, 1.591796875, -1.77734375, -0.6865234375, -0.0238494873046875, 1.5263671875, -0.336181640625, -0.40576171875, 0.74951171875, 1.439453125, 0.297119140625, -0.404052734375, -0.365966796875, -1.0380859375, 1.095703125, -0.12548828125, 0.5, -0.03314208984375, -0.1953125, -0.509765625, 0.048828125, -1.98828125, -1.02734375, 0.54833984375, 0.348876953125, -0.62353515625, -0.81103515625, 0.74169921875, -0.347900390625, -0.04742431640625, -1.5859375, 0.59619140625, -1.3798828125, -0.6171875, -1.7373046875, -0.71875, -0.1025390625, 1.2255859375, -1.263671875, 0.34912109375, -0.1572265625, -1.8701171875, -0.178466796875, 0.6787109375, 0.3740234375, 0.4072265625, 1.6123046875, -0.1768798828125, -1.224609375, 0.004116058349609375, 0.05975341796875, -0.8369140625, 1.7021484375, -0.1204833984375, -0.259033203125, 0.0748291015625, -0.287841796875, 1.544921875, -0.007289886474609375, -0.1851806640625, -1.001953125, -1.1064453125, -0.86376953125, -1.9970703125, 1.4501953125, 0.48974609375, 0.332763671875, -0.368408203125, -0.413818359375, -0.1436767578125, -1.3818359375, 0.9189453125, -0.8115234375, -1.7646484375, 0.1787109375, 0.95654296875, 0.421875, 1.408203125, -0.4443359375, 0.1822509765625, 0.80908203125, -0.247802734375, 1.4599609375, -0.190185546875, 0.01727294921875, -0.79931640625, 0.70556640625, 0.45751953125, -2.73828125, -0.07391357421875, -0.93115234375, 0.5322265625, -0.436767578125, 1.3818359375, 0.45703125, -0.51953125, 0.9599609375, -0.71484375, -0.26416015625, -0.91015625, -1.01953125, -0.51904296875, 0.634765625, -1.267578125, 1.28125, 0.86328125, -0.144287109375, 0.363037109375, -0.11474609375, 0.42578125, -2.49609375, 0.238037109375, -0.279541015625, 1.2470703125, 1.4306640625, 0.56591796875, -1.8623046875, 0.95166015625, -0.556640625, 1.41796875, 0.45947265625, 1.6591796875, -1.0751953125, -1.4560546875, -0.2196044921875, -2.470703125, -1.13671875, 1.0087890625, 1.4736328125, -1.638671875, 1.5859375, -0.1318359375, -1.208984375, 0.94775390625, -1.0478515625, -1.4521484375, 0.6142578125, 0.258544921875, -1.1611328125, -0.53515625, -0.72607421875, 0.9609375, 1.1064453125, -0.321044921875, 0.6572265625, -0.047027587890625, 0.170654296875, 1.068359375, 1.1083984375, -0.7353515625, -0.3408203125, -0.29736328125, 2.064453125, -0.7802734375, -1.0732421875, 0.413818359375, 0.342041015625, -1.3095703125, -2.826171875, 0.54248046875, -1.1083984375, -2.32421875, -0.458984375, -0.4384765625, 0.81591796875, -0.73583984375, 0.1888427734375, 0.237548828125, 0.28662109375, 1.892578125, 0.236083984375, -0.270263671875, -1.5126953125, 2.69921875, -0.019287109375, -0.0904541015625, 0.634765625, 0.06256103515625, 2.72265625, -2.0390625, -1.548828125, -1.013671875, -0.7978515625, -0.158203125, -1.3037109375, 0.8564453125, 0.160888671875, -1.322265625, 0.62353515625, 0.888671875, -0.056610107421875, -0.1278076171875, 0.498779296875, 0.73193359375, -0.98486328125, 0.96630859375, 0.1053466796875, 0.5400390625, 2.1640625, -0.419677734375, -0.438232421875, 1.109375, 1.0, 1.1337890625, 0.007537841796875, 0.1778564453125, -0.65966796875, -0.46240234375, 0.49755859375, 1.5537109375, 1.6005859375, -1.5703125, 0.62841796875, 0.6728515625, 0.62890625, 0.45849609375, 0.29541015625, -0.1434326171875, -0.93701171875, 1.4306640625, -0.76513671875, -0.334716796875, -0.424072265625, 0.0543212890625, 0.1302490234375, 0.208740234375, -2.08203125, -1.064453125, -0.80224609375, 0.658203125, -0.425048828125, 1.59765625, 2.974609375, -0.245361328125, 0.135498046875, -1.7177734375, -0.047271728515625, -0.34912109375, 0.31591796875, 0.0853271484375, 0.75146484375, -0.430908203125, 0.06695556640625, 0.1663818359375, -0.80078125, 2.404296875, 0.79296875, -1.3173828125, 0.78271484375, -1.5654296875, -0.56298828125, -0.481689453125, -0.60009765625, 0.287353515625, 1.0244140625, -1.28125, -1.513671875, -0.7587890625, -1.666015625, -1.5205078125, 0.65966796875, -0.07470703125, -0.79833984375, 1.109375, -1.10546875, -1.3623046875, 0.43359375, -0.73486328125, 0.7314453125, -0.89013671875, -0.71044921875, -0.9375, -1.431640625, -0.2744140625, -0.857421875, 0.3359375, 1.7578125, -0.0548095703125, -0.80859375, -0.86865234375, 0.06671142578125, -2.220703125, -0.1751708984375, 2.546875, -0.90185546875, -0.6474609375, -0.2398681640625, -0.74560546875, -0.421142578125, 0.6181640625, -0.0284881591796875, -2.072265625, -0.58447265625, 0.89404296875, -0.01441192626953125, -0.640625, 0.47607421875, 0.451416015625, 0.137939453125, 1.130859375, 0.288330078125, -1.447265625, -0.57080078125, -1.048828125, -0.225830078125, 0.740234375, -1.5810546875, 1.37109375, -0.9443359375, -1.4560546875, 0.962890625, -0.5361328125, -0.09710693359375, -0.0506591796875, -1.15234375, 0.82958984375, -0.2384033203125, 0.025238037109375, 1.048828125, -2.0390625, -0.52734375, -0.49755859375, 1.6748046875, 0.78271484375, -0.302734375, 0.7587890625, 1.1083984375, -1.12109375, 0.1531982421875, 0.6181640625, 0.16357421875, 0.0780029296875, -2.55078125, 0.59765625, -0.353759765625, -0.29638671875, 1.162109375, 0.11181640625, -1.955078125, -0.6474609375, -0.4599609375, 0.19580078125, 1.513671875, -0.96826171875, -0.86572265625, -0.25341796875, 1.7197265625, 0.65283203125, 0.767578125, 2.125, 0.2093505859375, -0.08221435546875, -0.2125244140625, 0.8701171875, -0.77197265625, -0.5068359375, -1.3720703125, 0.273193359375, -0.90087890625, -0.5849609375, 0.7744140625, -0.469970703125, -0.207763671875, 1.2451171875, 0.7021484375, 1.611328125, -1.466796875, 0.0278167724609375, 1.501953125, 1.16015625, -0.22509765625, 0.98291015625, -0.287109375, 0.053436279296875, -0.00927734375, 0.1890869140625, -1.2607421875, -1.4404296875, -0.4326171875, 0.342529296875, -1.4794921875, 0.497314453125, 2.4609375, 0.048980712890625, 0.72900390625, -0.62158203125, 1.125, -0.1754150390625, -1.0498046875, 0.11492919921875, -0.708984375, -0.11279296875, 0.841796875, 0.16162109375, 0.155517578125, 0.50830078125, 0.9287109375, 2.0859375, -0.7353515625, 1.0263671875, 1.0869140625, -0.09088134765625, 2.521484375, -0.43994140625, 1.0576171875, -0.130615234375, 0.2061767578125, -0.544921875, 0.1998291015625, 0.611328125, -0.71533203125, 0.2022705078125, 0.09771728515625, 1.9912109375, -0.658203125, 0.328369140625, -1.6904296875, 1.12890625, 0.412841796875, -1.1953125, 0.5400390625, 0.25390625, -0.2509765625, -0.341552734375, 0.370849609375, 0.042572021484375, -1.8349609375, 0.87548828125, -0.6103515625, -0.1025390625, 1.4736328125, 0.86376953125, -0.65087890625, 0.142333984375, -0.468994140625, 0.0809326171875, -1.2724609375, 0.2489013671875, 0.452392578125, -0.150634765625, -0.6875, 0.0289306640625, -1.853515625, -0.58984375, 0.37255859375, -0.443115234375, 1.263671875, -0.08599853515625, 1.037109375, -0.92578125, -0.088134765625, 1.0458984375, 0.65673828125, 1.6787109375, 0.383544921875, 0.74853515625, -0.07080078125, -0.05682373046875, -1.837890625, -0.0982666015625, 0.8740234375, 0.408447265625, -0.28466796875, 2.076171875, -1.6318359375, -0.06732177734375, 0.841796875, 0.01435089111328125, 0.88330078125, 0.51513671875, -1.3896484375, 0.007160186767578125, -0.5361328125, -1.1787109375, -0.1341552734375, -1.0146484375, 1.0947265625, -1.3818359375, -1.7509765625, 0.56591796875, 1.13671875, -0.315673828125, 0.44091796875, 0.7978515625, 1.5302734375, 0.034912109375, -1.1162109375, 0.0943603515625, 0.69140625, 0.98193359375, 0.301025390625, -0.83935546875, 1.619140625, 1.166015625, 0.5322265625, 0.73828125, 0.2398681640625, 0.54541015625, -0.4208984375, 0.323974609375, 0.257080078125, -0.6796875, -1.0693359375, -0.77294921875, 0.34375, -0.149169921875, 1.1611328125, -0.58740234375, -0.01348876953125, 0.1622314453125, 0.12255859375, 0.29150390625, -0.2493896484375, 0.99560546875, -0.255126953125, -0.6826171875, 0.056884765625, -0.02618408203125, -0.4462890625, 0.611328125, -1.1669921875, 2.314453125, 0.208984375, -0.71142578125, -1.2060546875, 0.25732421875, 1.82421875, -0.269287109375, -0.7216796875, -1.3310546875, -0.60498046875, 1.5615234375, 0.35791015625, 0.18115234375, 0.5927734375, 0.466064453125, 0.63330078125, -0.01381683349609375, -0.8544921875, -0.7626953125, -1.083984375, -2.6171875, -0.332763671875, -0.96826171875, 0.77001953125, -0.1893310546875, 0.20263671875, -0.024871826171875, -0.8876953125, -0.5322265625, 0.2088623046875, 1.0390625, 0.0070037841796875, -0.12200927734375, 1.3935546875, -1.154296875, 1.4150390625, -2.552734375, -0.016143798828125, 0.1392822265625, -0.1954345703125, 0.80615234375, -0.41943359375, 0.1390380859375, 2.837890625, -1.5654296875, 0.260498046875, 0.145751953125, 1.1630859375, 0.8798828125, -0.188232421875, -0.44775390625, -0.1639404296875, -1.3623046875, 0.311767578125, -0.78564453125, -0.062744140625, -1.1259765625, 0.11273193359375, 0.04510498046875, 0.35791015625, -0.00756072998046875, -0.78369140625, 0.2152099609375, 0.77490234375, -0.71875, -0.268798828125, -1.32421875, -1.1005859375, -0.489013671875, -0.029937744140625, 2.51171875, 1.76953125, -0.578125, -0.5556640625, 0.427734375, 0.716796875, -0.365234375, -0.58154296875, 0.65673828125, 0.14208984375, -0.72314453125, -0.56298828125, -0.1512451171875, 0.7978515625, 0.19091796875, 0.51123046875, 0.33447265625, 0.31982421875, 0.91650390625, 1.177734375, -0.413330078125, 1.630859375, -0.7431640625, 0.2254638671875, -0.158935546875, 1.44921875, -1.2646484375, 1.8583984375, -0.143798828125, -1.552734375, -0.29296875, 0.1522216796875, 0.449951171875, -0.33447265625, -0.9697265625, -1.1064453125, -0.205810546875, 1.783203125, -0.984375, -2.271484375, 0.8603515625, -0.427978515625, 1.0869140625, -1.96875, 0.1893310546875, 1.0634765625, 1.5927734375, -0.90673828125, -1.8447265625, -2.115234375, -0.041595458984375, -0.09405517578125, 1.5908203125, 0.6474609375, 0.146728515625, 0.176513671875, 1.4130859375, -0.5537109375, 0.970703125, -0.2119140625, 0.61279296875, -1.4560546875, 0.56201171875, -0.13330078125, 0.1697998046875, 2.173828125, -0.5009765625, -0.489501953125, 0.31103515625, -0.446044921875, -1.447265625, -0.72412109375, 0.76611328125, -1.095703125, -1.46875, 0.61865234375, -2.498046875, 1.228515625, 1.4365234375, -0.177001953125, -0.1632080078125, -0.32080078125, -1.5400390625, 0.91162109375, -1.51953125, -1.703125, -0.244873046875, 1.6806640625, -1.0625, -0.998046875, -0.00482940673828125, 0.7431640625, -0.4482421875, -1.2490234375, 0.56396484375, -0.6416015625, -0.49951171875, 0.93359375, -0.51708984375, 1.458984375, 0.0782470703125, 0.58642578125, 0.81005859375, 1.1787109375, 0.6484375, -0.355224609375, -1.091796875, 0.04620361328125, -0.399169921875, 0.87060546875, -0.6611328125, 0.11181640625, 0.59130859375, -1.697265625, -0.274169921875, 0.93212890625, -0.630859375, -1.068359375, -0.2880859375, 0.1129150390625, -1.1064453125, 1.11328125, 0.513671875, 0.447509765625, 0.452880859375, 1.4658203125, 0.890625, 2.26171875, -0.53466796875, -1.037109375, 1.03125, -0.189697265625, 0.68505859375, 0.6240234375, -1.4736328125, -1.1279296875, 3.099609375, 0.051239013671875, -0.80859375, -2.595703125, 0.29296875, -0.47802734375, -1.58984375, -1.484375, 1.7783203125, -0.22509765625, -0.3798828125, -1.79296875, -1.078125, -0.3896484375, -0.8740234375, 0.1767578125, 0.36767578125, -0.0131683349609375, -2.013671875, -0.640625, -1.01171875, 0.1666259765625, -0.99365234375, -0.318359375, -0.51953125, 0.72705078125, 0.411376953125, 0.2122802734375, -1.75390625, 0.31689453125, 1.294921875, 2.580078125, -0.8642578125, 0.1650390625, -0.77099609375, 0.59765625, 0.29638671875, 0.2509765625, -0.91845703125, -0.4375, -0.6669921875, -1.693359375, 0.05645751953125, -0.342041015625, -1.6572265625, 0.51708984375, 0.5205078125, -0.0919189453125, 0.59228515625, -0.892578125, -0.64208984375, 0.75, -0.78466796875, -0.50390625, 0.333984375, 0.677734375, 0.90869140625, -1.4912109375, 0.3671875, 1.837890625, 0.02069091796875, -1.70703125, 0.6591796875, -0.62548828125, 0.61181640625, 0.462158203125, -1.5107421875, 0.231201171875, -2.25, 0.32373046875, 0.8388671875, 0.1524658203125, -0.1314697265625, -1.1318359375, -1.1357421875, -0.199951171875, 1.697265625, 1.056640625, -0.52392578125, 0.0504150390625, 0.2431640625, 0.9873046875, 0.1300048828125, -1.9453125, 0.71435546875, 0.11187744140625, -0.9423828125, 2.83984375, -0.1385498046875, -1.3583984375, 0.0025386810302734375, -0.193359375, 0.76123046875, -0.294921875, 1.751953125, -0.2286376953125, 0.348876953125, 1.5, 2.41015625, -0.73388671875, -0.97216796875, -0.3994140625, 0.372802734375, 2.63671875, 0.49658203125, -1.6416015625, -0.1788330078125, 2.375, 0.033843994140625, 0.88330078125, -0.491943359375, -1.181640625, 0.107177734375, -0.14794921875, -1.521484375, 0.30712890625, 1.1455078125, 0.50390625, 0.5380859375, -1.275390625, -0.473876953125, 2.1171875, -0.1334228515625, -1.3056640625, -0.282470703125, 0.03875732421875, 1.3486328125, -0.9931640625, 0.943359375, 1.22265625, -0.1829833984375, -1.3505859375, 0.61083984375, 1.0419921875, -0.412353515625, 1.076171875, 0.1397705078125, 1.6630859375, -0.55029296875, -0.59130859375, -0.460205078125, -0.7392578125, -0.10565185546875, -0.49755859375, 1.65625, -0.478759765625, 1.322265625, 1.458984375, -0.89892578125, 0.6572265625, -0.3095703125, 0.42333984375, 1.2265625, -0.1646728515625, 0.349609375, -0.193115234375, -0.47119140625, -1.7744140625, 0.301513671875, 0.6435546875, 1.4150390625, 0.272705078125, -0.86669921875, -0.3310546875, -0.48828125, 1.3203125, 0.5810546875, -0.26220703125, -1.27734375, 0.411376953125, 0.20751953125, -1.5908203125, 1.083984375, -2.126953125, 0.794921875, -0.28369140625, 0.3232421875, -2.54296875, 1.935546875, -0.80908203125, -0.39404296875, -2.107421875, -0.1795654296875, -0.497802734375, 0.9140625, -0.418701171875, -0.56103515625, 0.84912109375, -0.9013671875, -1.625, -0.470458984375, 0.53076171875, 0.99072265625, -2.166015625, 0.62841796875, 2.62890625, 0.80859375, 1.5029296875, -0.099365234375, 0.320556640625, 0.50048828125, -1.3115234375, 0.90966796875, 0.29638671875, 1.0, -0.1968994140625, -0.09991455078125, 0.02630615234375, -1.328125, -1.18359375, 0.93994140625, -0.70068359375, -1.0595703125, 0.71728515625, 0.163818359375, 1.5556640625, -1.19921875, -0.82177734375, 1.4091796875, 0.72265625, 0.004528045654296875, 0.12841796875, -0.52099609375, -0.107421875, 0.477294921875, -0.978515625, -1.1923828125, 0.12396240234375, 1.1865234375, 0.73583984375, 0.42919921875, 0.2489013671875, 0.1719970703125, -0.9326171875, -2.35546875, 1.6279296875, 1.0341796875, 0.057952880859375, -0.323974609375, 1.5107421875, 0.5390625, 0.35400390625, 1.1298828125, -0.72900390625, -0.935546875, -1.021484375, -0.306396484375, -0.84033203125, 1.9638671875, -0.64453125, 1.30859375, -0.72607421875, -0.218017578125, -0.67578125, 0.0300445556640625, 1.0302734375, 1.517578125, 1.05859375, 0.07305908203125, 0.779296875, 0.70703125, -0.70751953125, -0.87255859375, -1.4326171875, -1.5517578125, -0.53076171875, 1.3662109375, -0.61083984375, -0.82470703125, 1.5419921875, 0.29150390625, -0.408447265625, -0.08819580078125, 1.4326171875, -0.0413818359375, 0.1395263671875, -2.591796875, 0.71630859375, -0.324462890625, 0.42529296875, 0.5087890625, -0.1126708984375, 0.1767578125, -1.115234375, -0.06951904296875, -0.732421875, 0.751953125, 0.60791015625, -0.1512451171875, -1.7900390625, 0.057281494140625, -1.0458984375, 2.18359375, -0.54248046875, -1.7021484375, -0.677734375, 0.207763671875, 0.6044921875, -0.947265625, -1.1943359375, 0.2156982421875, 1.2236328125, -2.533203125, 0.1488037109375, 0.75634765625, 0.1285400390625, -1.609375, -0.95947265625, -0.14990234375, 0.304443359375, 0.405029296875, 0.39306640625, -0.084228515625, -0.92919921875, 2.384765625, -0.1668701171875, -1.24609375, 0.95849609375, -0.61962890625, 0.0187835693359375, -0.88916015625, 1.2451171875, -0.5419921875, -1.134765625, 0.86572265625, -0.6015625, -0.331298828125, 1.056640625, 0.40283203125, 0.0289154052734375, 0.4521484375, -0.84228515625, -0.2186279296875, -1.6875, -0.58154296875, -0.96533203125, -1.34765625, 1.275390625, 0.15283203125, -1.49609375, -0.237060546875, -0.51611328125, -1.0146484375, 1.388671875, 0.385498046875, -0.288330078125, 1.375, 0.87890625, 0.423583984375, 0.142333984375, -0.73779296875, -0.51318359375, -0.310302734375, 0.390869140625, 1.4853515625, -0.23046875, -0.93408203125, 0.482177734375, -0.047119140625, 0.431640625, -0.1630859375, -0.685546875, 0.0194091796875, 0.312744140625, -0.78369140625, 0.63623046875, -0.282958984375, -0.310546875, 0.388427734375, -0.07666015625, -0.3212890625, 0.615234375, -1.166015625, 0.9443359375, -1.166015625, -0.35595703125, -0.17578125, 0.4716796875, -1.083984375, 1.3076171875, 1.5244140625, -0.30712890625, 0.92919921875, -1.2060546875, 0.9130859375, 0.273681640625, -2.083984375, 1.541015625, 0.272216796875, 0.77392578125, 0.01142120361328125, 0.007045745849609375, 1.078125, 0.33935546875, -1.3310546875, 1.25390625, 0.6318359375, 0.138671875, -0.66259765625, 0.06878662109375, -0.5322265625, -0.299560546875, 0.75634765625, 0.1649169921875, 0.64697265625, -0.191650390625, -1.392578125, -0.255126953125, -0.0836181640625, -0.2164306640625, 1.75, -0.1741943359375, 1.22265625, 0.042510986328125, -1.3544921875, -2.103515625, 0.2392578125, -0.09619140625, 2.107421875, 1.03515625, -0.0946044921875, 1.29296875, -0.246826171875, -0.1749267578125, -0.81591796875, -0.453125, 0.046661376953125, -0.058837890625, 0.8740234375, -0.11993408203125, -0.81396484375, -0.423828125, -2.009765625, 1.2666015625, 0.037200927734375, 1.1005859375, 0.3779296875, -0.5498046875, 1.2119140625, 0.13330078125, 0.438232421875, 1.0576171875, -0.1671142578125, -0.09375, 1.0546875, -1.3544921875, -0.1561279296875, -0.53759765625, 0.3759765625, 0.49462890625, 0.40234375, -1.5791015625, 1.736328125, -0.433349609375, 0.293212890625, -0.8251953125, 1.25390625, 2.205078125, 0.2861328125, 0.1282958984375, 0.08001708984375, -0.11578369140625, 0.5498046875, 1.607421875, 0.181884765625, -1.6201171875, 0.5400390625, -1.6767578125, 0.55419921875, -0.09112548828125, -0.50830078125, -0.7890625, 0.289306640625, 1.0888671875, 0.2313232421875, 0.607421875, -0.0161285400390625, 1.0224609375, -1.3330078125, 0.04388427734375, 0.01441192626953125, -1.73828125, -0.908203125, 0.3291015625, -0.26708984375, -1.8671875, 0.6875, 0.55908203125, 0.030181884765625, -0.057891845703125, -0.55517578125, -0.708984375, 0.3515625, -1.181640625, -0.032012939453125, -1.7607421875, 0.5947265625, 1.1025390625, 1.859375, 2.052734375, -0.01248931884765625, -0.96923828125, 1.4169921875, 0.0567626953125, -1.63671875, 1.1435546875, 0.5615234375, 0.2247314453125, 0.10626220703125, -2.203125, -0.246337890625, -0.326416015625, -0.732421875, -1.53515625, 1.4794921875, 0.0230865478515625, 0.92724609375, 0.48095703125, -0.11767578125, -0.491455078125, -0.6767578125, 0.8974609375, -1.279296875, -0.217041015625, 1.1630859375, 1.44140625, 0.1868896484375, -0.7548828125, 1.087890625, -1.083984375, -0.37939453125, 0.640625, 1.0693359375, 1.3095703125, -0.10418701171875, 1.64453125, -0.21533203125, -0.5810546875, -1.0224609375, -0.0054473876953125, 0.96875, -0.476806640625, -0.42333984375, 0.06646728515625, 1.1025390625, 1.8828125, 1.4033203125, -0.58349609375, -0.305419921875, 1.4248046875, -1.1171875, -1.0673828125, 1.8076171875, -0.64111328125, 1.7666015625, 0.4990234375, 0.1915283203125, 0.2032470703125, -0.34423828125, -0.03546142578125, 0.974609375, 0.4873046875, -0.58984375, 0.41748046875, 0.98291015625, -0.1446533203125, 0.33447265625, 0.76806640625, 0.80615234375, -0.0278167724609375, 0.0653076171875, -0.5009765625, 0.5517578125, 1.3701171875, -0.485595703125, -0.4150390625, -0.40234375, 0.261962890625, -1.4248046875, 0.275390625, 0.419189453125, 0.286376953125, -1.775390625, 0.195556640625, -0.326171875, 0.408203125, -0.5458984375, 0.65283203125, 0.822265625, 0.80029296875, -0.8388671875, -1.0537109375, -0.43798828125, -0.0650634765625, 2.2578125, 2.44140625, -0.68505859375, -1.306640625, 0.2337646484375, -0.8388671875, -2.615234375, 0.9912109375, -0.32373046875, 0.939453125, 0.2462158203125, 0.6904296875, 0.195068359375, -0.5556640625, 1.4814453125, -0.093017578125, 0.83642578125, -1.150390625, -0.475341796875, 0.3505859375, -2.298828125, 0.5341796875, -1.0146484375, -0.2369384765625, 0.9794921875, -0.312255859375, 0.295654296875, 0.453369140625, 1.3505859375, -0.85205078125, -0.046600341796875, -1.16796875, -0.9228515625, -1.486328125, -1.048828125, 0.59619140625, 0.400390625, -0.006671905517578125, -0.72021484375, 0.60009765625, -0.146484375, 1.0341796875, 0.045989990234375, 0.1180419921875, -0.390380859375, 0.334228515625, 0.2369384765625, -0.430419921875, -0.11688232421875, 0.189208984375, 0.26611328125, 0.55712890625, -0.0298004150390625, -0.58935546875, -0.92919921875, 0.446044921875, -2.193359375, -0.8056640625, 0.06793212890625, 0.708984375, 0.69921875, -0.050537109375, 1.9072265625, 1.7646484375, -1.017578125, -1.5166015625, 0.252197265625, -1.9423828125, -1.345703125, -0.458251953125, -0.196044921875, -0.90283203125, -0.7705078125, -1.158203125, 1.56640625, -1.263671875, -0.916015625, -0.521484375, 0.8505859375, -0.76953125, 2.126953125, 1.3876953125, 0.2325439453125, 0.7763671875, 0.4150390625, 0.9423828125, 0.5205078125, 2.220703125, -0.61181640625, -1.0498046875, 0.336181640625, 1.1240234375, -0.217041015625, 0.19970703125, -3.13671875, -1.3173828125, -0.98779296875, -1.0927734375, -0.869140625, -0.259521484375, -0.444580078125, 0.58740234375, 0.5078125, -1.2236328125, 0.362060546875, -0.6943359375, 0.626953125, 1.724609375, -0.2105712890625, 0.353759765625, 0.82421875, -0.054718017578125, -1.2392578125, -1.208984375, 0.15771484375, -0.71337890625, 0.376220703125, -3.2109375, -0.10650634765625, 0.325927734375, 1.6630859375, -0.473876953125, 1.6181640625, 0.125732421875, 0.57568359375, -0.92724609375, 1.13671875, -0.84619140625, 1.37109375, -0.970703125, -1.4423828125, -0.7392578125, 0.031280517578125, -0.5185546875, 0.90283203125, -0.9794921875, 0.7255859375, 1.2138671875, -1.3203125, -1.12109375, -1.103515625, -1.0205078125, -0.896484375, 1.755859375, 1.6201171875, -0.53076171875, 1.0419921875, -0.5126953125, 0.22509765625, -1.1015625, 1.578125, 0.054229736328125, 0.59130859375, 0.421630859375, 0.01099395751953125, 1.607421875, -0.59765625, 1.7822265625, -1.4677734375, 1.31640625, -0.7607421875, -2.6953125, 0.4091796875, 0.4267578125, -0.031494140625, 0.1334228515625, 0.39111328125, -0.6982421875, 0.037994384765625, -0.517578125, -2.521484375, -0.77783203125, 0.1759033203125, -0.5947265625, -0.1455078125, 2.388671875, 0.53759765625, 1.306640625, -1.6298828125, -0.953125, 1.439453125, 0.57470703125, -0.69775390625, 0.6494140625, -1.0224609375, 1.8203125, 0.69140625, 0.37890625, 0.26708984375, -1.357421875, -2.26953125, -0.572265625, 0.6357421875, -0.79052734375, -0.8232421875, 0.484130859375, 0.0462646484375, -0.66064453125, 0.1346435546875, 0.425048828125, 0.314208984375, 0.27001953125, 0.1385498046875, 1.833984375, 1.173828125, 0.94140625, 1.5283203125, -1.0224609375, -0.1832275390625, -0.058441162109375, -0.254150390625, -0.873046875, 0.7099609375, 0.2135009765625, 0.69580078125, -0.12042236328125, -0.271728515625, -0.200927734375, -0.5068359375, 1.005859375, 1.1201171875, 0.3447265625, -0.83642578125, -1.640625, 0.58154296875, -0.164794921875, -1.2568359375, -2.310546875, 0.0625, 1.126953125, -2.330078125, 0.11419677734375, 0.9765625, 0.048614501953125, -0.84716796875, 1.4697265625, 1.46875, 0.1826171875, -3.642578125, 1.09765625, -1.5234375, 0.72412109375, -0.26513671875, -1.35546875, 0.82470703125, -1.869140625, 1.0419921875, 0.95458984375, 0.54052734375, -0.8955078125, 0.9755859375, -0.27392578125, 0.5166015625, 0.0148773193359375, 0.421875, -1.0849609375, 0.245361328125, -0.370849609375, -1.13671875, 0.09527587890625, -0.78125, 2.115234375, 0.444091796875, 0.1658935546875, 0.5966796875, 0.135009765625, 1.23046875, 0.435546875, 0.021392822265625, -0.5205078125, 0.025115966796875, 0.6708984375, -2.107421875, -2.26171875, -0.1707763671875, -1.951171875, -0.8583984375, 0.457763671875, -1.9462890625, 0.65283203125, 1.318359375, 0.60205078125, 0.55126953125, -1.142578125, -0.68505859375, -2.91015625, 0.5966796875, 0.1041259765625, -1.4052734375, 0.87744140625, -2.0546875, -1.4951171875, -0.53955078125, -0.517578125, -1.2314453125, -0.438232421875, 1.3525390625, -0.5048828125, -0.289306640625, 0.43603515625, -0.61474609375, 0.34375, 0.491455078125, 1.068359375, 1.220703125, 0.73828125, -0.09893798828125, -1.1865234375, 0.34228515625, 0.087158203125, -1.1025390625, -0.578125, 1.9677734375, 0.97607421875, -0.147216796875, 1.9052734375, 0.9150390625, 0.32568359375, 0.323974609375, -1.6279296875, 0.74560546875, 0.476318359375, 0.28857421875, 0.1153564453125, 1.7919921875, -0.5458984375, -1.20703125, 0.93603515625, -0.80810546875, 0.38720703125, -2.2265625, -1.318359375, 0.14306640625, 0.8994140625, -1.24609375, 0.6982421875, -0.35009765625, 0.9150390625, -1.6640625, -0.28271484375, 0.75830078125, 0.1947021484375, 0.8173828125, 0.796875, -0.79345703125, 0.72998046875, -0.890625, -1.4091796875, 1.50390625, 0.7998046875, 0.146484375, -1.0185546875, -0.291259765625, -0.89599609375, 1.083984375, -0.2161865234375, 0.94775390625, 0.04425048828125, -0.32568359375, -1.4423828125, 1.4462890625, -1.5166015625, 0.1951904296875, -0.7451171875, 0.0609130859375, -0.59326171875, 1.3427734375, 1.2705078125, 0.73095703125, -1.85546875, -0.97412109375, -0.29638671875, 0.489013671875, -1.0341796875, -0.6767578125, 0.05419921875, -0.6904296875, -0.79736328125, 1.03125, -0.1561279296875, 0.88623046875, 0.69970703125, 0.5048828125, -0.732421875, 0.004764556884765625, 0.09600830078125, -1.458984375, -0.447998046875, -0.273193359375, 0.8779296875, 0.058135986328125, -0.296875, 0.3271484375, 0.407958984375, 0.490234375, 0.61279296875, -0.712890625, 0.07196044921875, 0.1385498046875, 0.2232666015625, 1.3203125, 0.0267181396484375, 0.77880859375, 0.487548828125, 1.8837890625, -0.54443359375, -0.8515625, 0.28466796875, -1.41796875, -0.8779296875, -1.6689453125, -0.356689453125, 1.693359375, -1.4111328125, -0.051055908203125, -1.634765625, -1.021484375, 2.40625, 0.1602783203125, 1.146484375, 0.63623046875, -1.9384765625, 0.76171875, -1.185546875, -0.287353515625, 0.2498779296875, 0.43896484375, 2.09375, -0.53173828125, 0.0113677978515625, -0.63916015625, 0.2247314453125, 0.3388671875, -0.0684814453125, 1.265625, 0.0831298828125, -0.73193359375, 0.161376953125, 0.607421875, 0.78515625, -0.1641845703125, -1.1875, -1.6318359375, -0.62548828125, 0.5654296875, -1.7412109375, -0.6298828125, 0.419189453125, -0.63623046875, -0.77099609375, 2.15625, -0.5947265625, 1.0556640625, 0.81640625, -0.285400390625, 0.5244140625, -0.6416015625, -2.1796875, 1.1708984375, 0.06109619140625, 0.53564453125, 1.7734375, -0.33203125, -0.03302001953125, -0.407958984375, -0.1448974609375, 0.0714111328125, -0.152587890625, 0.705078125, 0.1422119140625, 0.0274200439453125, -1.1787109375, 0.11065673828125, -0.453857421875, -0.461669921875, 0.497802734375, -0.27392578125, 1.0107421875, -0.11322021484375, -0.188232421875, -0.62451171875, 0.2213134765625, -1.34765625, 0.39453125, -0.1258544921875, -0.25390625, -1.3779296875, 0.599609375, 1.2666015625, -0.499267578125, -0.9052734375, -0.38037109375, 0.865234375, 0.54248046875, -0.7412109375, 0.68994140625, 0.9443359375, -1.0263671875, -0.129150390625, -0.61865234375, -0.587890625, -0.76806640625, -0.4052734375, 0.2384033203125, 0.1278076171875, -0.85009765625, 1.0927734375, 0.486572265625, -0.464599609375, -1.4443359375, -0.90234375, 1.2060546875, 0.66796875, 1.107421875, -0.69677734375, 0.673828125, 0.5048828125, -0.85595703125, -1.005859375, -0.537109375, 0.35693359375, 1.4609375, -0.64306640625, 1.0146484375, -0.51953125, 0.496337890625, -0.62890625, -0.78955078125, -1.458984375, -0.284423828125, -0.92578125, -0.56298828125, 0.248291015625, -2.220703125, 0.1668701171875, 0.308837890625, -0.0926513671875, 0.79443359375, 0.39697265625, -0.058868408203125, 1.4208984375, -0.2958984375, -0.1500244140625, -0.226806640625, -0.7333984375, -0.806640625, 1.5302734375, 0.74755859375, -0.0821533203125, 1.2890625, 1.271484375, -1.4296875, -0.65771484375, 0.482177734375, -0.175048828125, 0.498046875, 0.29931640625, 2.681640625, -0.88330078125, -1.3134765625, 1.2841796875, 0.27001953125, 1.0771484375, -1.580078125, -1.685546875, -0.7646484375, 2.724609375, 0.08099365234375, -1.24609375, 0.8662109375, 0.38525390625, -1.4072265625, -0.126220703125, 0.9912109375, 1.431640625, -0.97265625, 0.8115234375, 0.451904296875, 0.233642578125, 0.5849609375, -0.2376708984375, 0.9716796875, -1.0654296875, -0.54150390625, 0.4951171875, 1.4365234375, 0.7275390625, -0.5, 0.75341796875, -0.334228515625, 0.37744140625, -1.3974609375, 2.08984375, 0.07147216796875, 1.44140625, 1.146484375, -0.1434326171875, 0.57470703125, -0.20263671875, 1.115234375, 0.131103515625, -1.2646484375, 0.8974609375, 1.2158203125, 1.0986328125, -0.1904296875, 0.32568359375, -2.17578125, -0.41357421875, 0.732421875, 1.6015625, 0.53466796875, -1.3974609375, -1.076171875, -0.032196044921875, -0.268310546875, 1.2421875, -1.08203125, 0.5595703125, 1.3486328125, -0.86181640625, 1.943359375, 0.2471923828125, 0.0254669189453125, 0.62744140625, 0.0618896484375, -2.130859375, 1.373046875, 1.0810546875, -0.12548828125, 2.486328125, -1.232421875, 0.6787109375, -0.33251953125, 0.6845703125, -0.73291015625, 1.6455078125, 1.0146484375, 0.390625, -0.2462158203125, 0.15283203125, -0.57275390625, -0.1632080078125, 0.1239013671875, -1.751953125, -0.54052734375, -1.80859375, 1.2880859375, 0.175537109375, 1.048828125, -0.196533203125, 1.5732421875, 0.73974609375, -2.044921875, -0.328125, -0.229736328125, 0.058135986328125, 0.5322265625, 0.024810791015625, 2.169921875, 1.9462890625, -0.5478515625, 0.21630859375, 0.70458984375, 0.48291015625, -0.132080078125, 0.70947265625, 0.492431640625, -0.0010595321655273438, 0.80517578125, 0.41650390625, -0.252685546875, 0.037872314453125, -1.3896484375, 0.501953125, -1.974609375, 0.83984375, 0.2320556640625, 0.1414794921875, 2.021484375, 0.4921875, -1.01171875, 0.67578125, 0.70068359375, 0.2802734375, -0.74658203125, 1.1943359375, -1.478515625, -0.394287109375, -1.0234375, 1.8828125, -0.82421875, 0.1348876953125, -1.34765625, 0.463623046875, 0.284423828125, -0.52978515625, -0.748046875, -0.1893310546875, 1.7119140625, 0.671875, -2.04296875, 1.0986328125, -1.50390625, -2.5390625, -1.103515625, 0.03118896484375, -0.2474365234375, -0.033355712890625, -0.67236328125, 0.71435546875, 0.2359619140625, 0.123779296875, -0.71484375, -0.72998046875, -0.97412109375, -0.59228515625, 0.7138671875, 0.68310546875, 0.49853515625, -1.5361328125, -1.3720703125, -1.044921875, 0.8251953125, 1.29296875, -0.320068359375, -1.224609375, -0.406494140625, -1.521484375, 0.487548828125, 0.52392578125, -1.7177734375, 0.231201171875, -0.330322265625, -0.7333984375, -0.1854248046875, -1.521484375, -0.375, -0.27734375, -0.33642578125, 0.258056640625, -1.83984375, 2.162109375, 0.60546875, 1.623046875, 0.0192108154296875, 0.0391845703125, 1.4287109375, -0.42138671875, -1.185546875, 2.080078125, 1.8310546875, 0.5341796875, -0.548828125, -1.0673828125, -0.51416015625, -0.630859375, -0.86376953125, -0.189697265625, 0.402587890625, 0.59521484375, 0.36767578125, 0.260009765625, -0.23193359375, -1.0146484375, 0.60888671875, -1.326171875, -2.1796875, 0.1195068359375, -2.51171875, 1.212890625, 1.0078125, 0.5908203125, -0.38525390625, 0.343505859375, -0.06719970703125, 0.288330078125, 0.1566162109375, 1.326171875, 1.3583984375, 0.31103515625, -0.5869140625, -1.26171875, 0.86767578125, 0.51025390625, -0.26416015625, -0.8857421875, 0.1795654296875, 0.94482421875, -0.178955078125, -0.41943359375, 0.201171875, 0.2142333984375, -0.60595703125, -0.6689453125, -0.0924072265625, 0.591796875, 0.4501953125, -0.2244873046875, 1.16015625, -0.50244140625, 1.4833984375, 1.021484375, -0.49755859375, 0.58984375, -0.7041015625, 0.54296875, 1.015625, -0.258544921875, 0.8798828125, -0.0980224609375, -0.90673828125, 0.379150390625, -1.078125, 1.0869140625, 0.78515625, -0.371826171875, 0.30810546875, -0.297119140625, -0.17822265625, 1.650390625, 0.59130859375, 0.284912109375, 1.91796875, -0.2171630859375, -0.5478515625, -2.072265625, -0.6611328125, 0.468505859375, -0.7265625, 0.1534423828125, -0.40673828125, 0.255859375, -2.33203125, -1.47265625, -1.00390625, 1.6572265625, 0.80712890625, -0.0145416259765625, 1.2197265625, -1.9111328125, -0.91259765625, 1.60546875, -1.3916015625, 1.162109375, 0.89599609375, -1.4794921875, 0.019195556640625, 0.339599609375, 1.2861328125, 0.127685546875, 0.27099609375, -0.8505859375, -1.5673828125, 2.796875, 0.040283203125, -0.50537109375, 0.82177734375, 0.8779296875, -1.197265625, 0.8173828125, -0.15771484375, -1.0869140625, -1.021484375, 0.7666015625, 1.0654296875, 0.67919921875, -0.92626953125, -0.201904296875, -1.240234375, -0.435791015625, -0.483642578125, 0.78173828125, 0.48388671875, 0.509765625, 1.5380859375, -0.580078125, -0.374267578125, -0.009002685546875, 1.078125, 0.51904296875, -0.3701171875, 1.1669921875, -0.93310546875, -0.9912109375, -0.2568359375, -0.2061767578125, 0.12066650390625, 0.1259765625, -0.0059051513671875, -0.8740234375, -2.927734375, -0.322509765625, -0.61083984375, 0.2177734375, -1.333984375, 0.0487060546875, -0.11273193359375, -0.2161865234375, 1.5947265625, -0.193359375, 0.5537109375, -1.0263671875, -0.62158203125, -0.0931396484375, 1.5615234375, -0.169677734375, -1.1494140625, 1.494140625, -2.1328125, 0.71533203125, 0.335205078125, 0.4345703125, -0.87646484375, -1.111328125, 1.2841796875, 0.80859375, 0.259521484375, 0.047271728515625, -0.31298828125, -1.013671875, -0.744140625, 0.281005859375, -1.869140625, -2.12109375, -1.806640625, 1.7646484375, -1.169921875, 0.52197265625, -0.227294921875, -1.5126953125, 2.30078125, 0.2322998046875, 0.84912109375, 0.04718017578125, -0.52392578125, -1.666015625, -0.57421875, 0.599609375, 0.5390625, -0.10736083984375, -1.3935546875, -0.75732421875, -0.50634765625, -0.1024169921875, 0.7119140625, 1.1376953125, -0.73583984375, 0.6640625, -0.0574951171875, 0.121337890625, -0.70458984375, 0.701171875, -0.693359375, 1.2490234375, -0.39111328125, -1.5625, -0.46484375, 1.955078125, 0.1488037109375, 0.1177978515625, -2.322265625, -0.08941650390625, -0.6884765625, 2.27734375, -0.77294921875, -0.943359375, -0.1778564453125, 0.0999755859375, 0.9111328125, 0.1029052734375, -0.96240234375, 0.56201171875, -0.3486328125, -0.255126953125, 1.8935546875, 0.8046875, -0.671875, -0.2108154296875, 0.075927734375, 0.546875, 0.056396484375, 1.1005859375, -0.65283203125, 1.197265625, -0.205078125, -0.2548828125, -1.5146484375, 1.4521484375, -0.332275390625, -1.2099609375, 0.055084228515625, -0.978515625, -1.1962890625, -0.51904296875, -1.15625, 1.15625, 2.125, -0.55615234375, -0.78076171875, 1.5986328125, 0.0823974609375, 0.022064208984375, -0.79150390625, -2.490234375, -0.9326171875, -0.208984375, 0.30078125, -0.08978271484375, 0.350830078125, -0.56640625, -0.63525390625, -1.5849609375, -1.3994140625, 0.08111572265625, -0.95947265625, 0.595703125, 1.107421875, 0.343017578125, -0.51025390625, 0.94873046875, 0.38623046875, 0.47900390625, -1.0146484375, -1.173828125, -0.875, -1.29296875, -0.06854248046875, 0.630859375, -2.22265625, 0.058837890625, 0.1668701171875, -0.24267578125, 1.75, 1.4892578125, -0.88134765625, 0.1495361328125, -0.93212890625, -2.021484375, 1.03125, -1.1279296875, 0.97998046875, -0.1124267578125, -0.51611328125, 0.97900390625, -1.513671875, 0.7626953125, 1.5947265625, -1.412109375, 1.484375, -0.424072265625, 0.435302734375, -0.0712890625, -0.57080078125, -0.787109375, -0.11700439453125, 1.142578125, -1.1015625, 0.50390625, -1.0087890625, -0.853515625, 0.27099609375, -1.1103515625, -0.2464599609375, 0.057525634765625, 0.3046875, 0.24072265625, 0.39599609375, 0.62939453125, 0.51171875, -1.1982421875, -1.072265625, -0.826171875, 0.6083984375, 0.1700439453125, 0.332763671875, -1.2080078125, -1.6943359375, 1.3544921875, -0.41552734375, -1.2177734375, 0.8212890625, -1.240234375, 0.53955078125, -1.744140625, -1.1943359375, -0.5361328125, 0.71044921875, -0.066162109375, 0.10443115234375, -0.1290283203125, -0.72705078125, 0.76123046875, 0.2423095703125, -1.6337890625, 0.05987548828125, -1.1689453125, -0.43603515625, -1.0078125, 0.62841796875, -1.3779296875, 0.06640625, 0.386474609375, 0.0838623046875, -0.75341796875, -0.2919921875, 0.196533203125, -0.30078125, -0.8037109375, 0.53564453125, 0.103759765625, -1.5498046875, -0.07470703125, -0.7080078125, 0.335205078125, 0.1943359375, 0.146484375, 1.16796875, -0.2396240234375, 0.483154296875, 0.4287109375, -1.23046875, -0.423095703125, -2.779296875, -1.3525390625, -1.79296875, -1.2353515625, -0.083251953125, -1.2919921875, -0.3505859375, 1.7216796875, 0.90771484375, -0.93798828125, -0.269775390625, -0.4345703125, 0.421875, -0.9814453125, 1.12890625, 0.74365234375, 0.5126953125, 0.60107421875, 1.984375, 0.896484375, -0.492431640625, -0.2464599609375, 2.37890625, -1.099609375, -0.5029296875, -0.56005859375, 0.32275390625, -0.1395263671875, -1.8408203125, 0.82421875, -0.475341796875, 0.11181640625, -1.6240234375, -1.1181640625, 0.677734375, -0.490966796875, -1.1484375, 1.744140625, -0.479248046875, 0.405029296875, 0.8818359375, -0.0029392242431640625, -0.07904052734375, 0.36279296875, 0.8798828125, 0.056915283203125, 2.169921875, -0.0338134765625, 0.2099609375, -0.1561279296875, 0.2154541015625, 0.0106201171875, 0.5, -1.3935546875, 1.1767578125, 1.2724609375, 0.6279296875, 0.5458984375, -0.322509765625, -1.142578125, 0.371337890625, -0.67919921875, -1.380859375, -0.50537109375, -0.33203125, -1.9912109375, -0.58935546875, 0.17822265625, 0.91259765625, -0.93017578125, 0.53125, 1.94140625, 0.384521484375, 1.6611328125, 0.67724609375, 0.0038547515869140625, -1.025390625, 0.33544921875, 1.4375, 0.62060546875, 0.6875, 0.049163818359375, -0.640625, -1.0732421875, -0.572265625, 1.4228515625, 0.257568359375, 1.5703125, 1.994140625, -0.63427734375, 0.415771484375, 0.3857421875, 0.4130859375, 0.99609375, -1.6279296875, 0.9638671875, -0.06231689453125, -1.0322265625, -1.5927734375, -1.2841796875, 0.5224609375, -0.438720703125, -0.69482421875, 0.4482421875, -0.483154296875, -0.21923828125, 1.2529296875, 0.880859375, -0.4091796875, 1.298828125, 1.21484375, 0.87353515625, 1.388671875, 0.06591796875, -0.7705078125, 0.11083984375, -1.146484375, 2.041015625, 0.210693359375, 2.0546875, 1.595703125, 0.748046875, -1.193359375, 0.8974609375, 0.677734375, 0.3154296875, 1.4755859375, -1.46484375, -1.2998046875, 1.0185546875, -0.388916015625, 0.86669921875, 0.46484375, 1.4716796875, -0.68408203125, 1.4208984375, -0.053497314453125, -0.1318359375, -0.5498046875, 0.74267578125, -0.76708984375, 0.88671875, -1.234375, 0.42041015625, -0.07818603515625, -0.29931640625, -0.67919921875, -0.487060546875, -0.26416015625, 1.001953125, 0.351318359375, -1.22265625, -2.529296875, 0.308837890625, -1.0986328125, 0.453369140625, 0.039337158203125, 0.7158203125, 0.06805419921875, 0.697265625, -0.26416015625, -0.50146484375, 1.00390625, -0.5703125, 0.349853515625, 0.190185546875, 1.0185546875, 1.64453125, -0.72509765625, -1.7421875, -0.70263671875, -0.2418212890625, 0.7548828125, 0.56396484375, 0.64697265625, -0.85107421875, -0.61181640625, 1.9091796875, -1.2041015625, -0.35205078125, 0.81884765625, -1.015625, 0.69677734375, -1.830078125, 0.142822265625, -0.009246826171875, -0.02001953125, -1.67578125, 2.1484375, -0.6845703125, 0.306884765625, -0.152587890625, 0.75244140625, 0.272216796875, 0.1689453125, 1.0712890625, -0.007732391357421875, -0.90283203125, 1.220703125, 1.654296875, -2.45703125, -1.513671875, -0.83740234375, 1.2451171875, -1.3984375, 0.326171875, -0.462646484375, 0.1619873046875, -1.51171875, -0.6123046875, 1.7822265625, -0.73828125, -0.62255859375, 1.4541015625, 1.388671875, -0.38330078125, -1.1611328125, 0.05047607421875, -1.61328125, -1.1494140625, -1.02734375, -1.6845703125, -1.64453125, 0.97265625, -0.1312255859375, -0.74853515625, 0.0242462158203125, 0.24560546875, 1.65625, 0.73828125, -0.5947265625, -0.76318359375, -0.461181640625, -1.9248046875, 0.44775390625, 1.6357421875, 1.5849609375, -1.02734375, -1.07421875, 1.369140625, -1.4736328125, -1.0107421875, -0.328125, -0.402587890625, -0.1810302734375, -0.498779296875, 0.37939453125, 0.28125, -0.58984375, 0.8212890625, 0.578125, -1.400390625, -0.03857421875, 0.35791015625, 0.0038909912109375, 0.2216796875, 0.188720703125, -0.54345703125, -0.1015625, -0.7255859375, -0.0318603515625, -0.339599609375, -1.16796875, 0.1964111328125, -0.389892578125, -1.3291015625, 0.479248046875, -0.104248046875, -0.036590576171875, -1.0869140625, -0.276611328125, -2.02734375, -0.55224609375, -0.9541015625, 2.244140625, -1.3203125, -1.2626953125, 0.2161865234375, -0.7900390625, -0.52294921875, 0.06829833984375, 0.80078125, 0.497314453125, -1.3505859375, -1.0869140625, -0.708984375, -0.71435546875, -0.65478515625, -1.572265625, 0.72607421875, 0.78515625, 0.74169921875, -0.451171875, -2.7734375, 0.892578125, 0.91162109375, 1.15234375, 0.0565185546875, 0.4150390625, -0.371826171875, 0.79443359375, 0.324951171875, -0.12274169921875, 1.3251953125, 2.15625, -0.2078857421875, -1.5048828125, 1.4072265625, -0.130126953125, 0.75732421875, -0.79443359375, 0.9990234375, -0.001087188720703125, -1.283203125, -1.8349609375, -0.5791015625, -1.1064453125, 0.06964111328125, 1.48046875, 0.919921875, -0.282958984375, -0.541015625, -0.705078125, 2.623046875, -1.5234375, 0.0262451171875, 0.37890625, -0.61181640625, 0.0030803680419921875, -0.55029296875, 0.85009765625, 1.548828125, -1.3095703125, 1.03515625, -0.78515625, 0.400634765625, 1.576171875, -2.115234375, -1.13671875, 0.062744140625, 0.97900390625, -1.25390625, 0.6806640625, 0.53955078125, -2.30078125, -0.30859375, -0.81591796875, 0.56884765625, -1.556640625, -0.08099365234375, 1.2158203125, -1.046875, 0.87451171875, -0.1533203125, -0.5234375, 1.296875, -0.84912109375, 0.2047119140625, 0.09674072265625, 1.1201171875, -0.254150390625, 0.8154296875, 0.0926513671875, -0.403564453125, 1.376953125, 0.08978271484375, -0.81640625, 0.68994140625, -0.46826171875, -1.3564453125, -0.036224365234375, -0.059356689453125, 0.387451171875, 1.8369140625, 0.396240234375, 0.186767578125, 0.44677734375, 0.626953125, -0.12091064453125, -0.88720703125, -0.6806640625, 1.11328125, -0.400634765625, 0.1702880859375, 0.006923675537109375, -0.488525390625, -0.72265625, -1.8798828125, 0.921875, 1.2763671875, 0.82958984375, 0.67236328125, 0.93017578125, -1.5888671875, 1.2041015625, 0.9794921875, -0.43017578125, 0.98779296875, -0.90771484375, -0.23291015625, 0.198974609375, -0.669921875, 1.166015625, 0.51708984375, 0.89990234375, -1.42578125, 1.5556640625, 1.498046875, -1.4833984375, 0.1778564453125, -1.2958984375, 1.5888671875, 1.1796875, 0.78125, -0.634765625, -1.46875, -0.86181640625, 1.3837890625, 1.7099609375, 0.7705078125, -1.408203125, 0.444091796875, -2.1015625, -0.951171875, -1.1708984375, -0.2354736328125, 0.06243896484375, 0.46240234375, 1.978515625, -0.56982421875, -0.82470703125, 1.2197265625, -0.64111328125, -1.4814453125, 0.55810546875, 0.3427734375, 0.059783935546875, -0.0138092041015625, -1.1474609375, 0.0679931640625, 1.560546875, -0.51611328125, 0.97998046875, -2.150390625, -0.5361328125, -0.93505859375, 0.050018310546875, -0.32080078125, 0.576171875, 1.392578125, -0.86572265625, 0.08526611328125, 0.109375, -1.5439453125, 0.529296875, 1.0009765625, -1.1787109375, 0.340576171875, 0.448974609375, 0.64697265625, -1.7587890625, -1.046875, -0.007476806640625, 0.56640625, 0.33349609375, -0.8837890625, 1.1767578125, 0.46630859375, -0.1265869140625, -1.54296875, -1.4970703125, -0.68017578125, -0.6279296875, 1.3349609375, 0.7568359375, 0.34619140625, -0.398681640625, -0.7939453125, -1.0576171875, 0.40283203125, -0.7763671875, 0.0477294921875, -0.0927734375, -0.366455078125, -1.64453125, -0.81591796875, -0.158935546875, -1.556640625, 0.1734619140625, 0.51806640625, 0.444091796875, -0.148193359375, -2.45703125, -0.52783203125, 1.6279296875, 0.7578125, -0.646484375, 1.19921875, -0.83740234375, -1.4814453125, -1.2861328125, 0.2384033203125, -0.44775390625, 1.90234375, -1.4326171875, -0.2802734375, 1.38671875, -0.370361328125, 1.5166015625, -1.478515625, 0.1868896484375, 0.6796875, 1.755859375, 0.6552734375, 1.4912109375, 1.4013671875, 0.050018310546875, -0.5302734375, -0.4189453125, -1.1513671875, -0.220703125, 0.347900390625, -1.2021484375, -1.8056640625, 1.5947265625, 0.11383056640625, 0.293212890625, -1.373046875, -1.443359375, -0.8408203125, 0.3134765625, 0.64990234375, -0.826171875, -0.681640625, 0.2364501953125, -0.055816650390625, -0.35546875, 0.4140625, -0.822265625, 0.84716796875, 0.403076171875, -1.7021484375, 0.9033203125, -0.47119140625, -0.4736328125, -0.5166015625, -0.2381591796875, 0.05126953125, 0.382080078125, -0.402099609375, 0.71337890625, 1.1240234375, -0.55419921875, 0.93408203125, 1.0849609375, 2.767578125, 1.224609375, -0.8349609375, -0.728515625, -1.931640625, 1.224609375, 0.1015625, 0.9560546875, -1.15234375, -0.84033203125, 0.44287109375, -0.1341552734375, 0.6865234375, 1.328125, 0.67138671875, 1.2626953125, 0.70751953125, -0.277587890625, 0.007175445556640625, -0.495849609375, -0.7861328125, -2.11328125, -0.75927734375, -0.47900390625, -0.056182861328125, -0.92138671875, -0.098876953125, -0.08428955078125, -0.2294921875, -1.0439453125, 1.00390625, -0.052703857421875, 0.7734375, 1.5400390625, 1.791015625, -1.736328125, 0.630859375, 1.5966796875, -1.87890625, -1.193359375, -0.291015625, 0.765625, -0.2261962890625, 1.8232421875, -0.66845703125, 0.48779296875, 0.353759765625, 0.11297607421875, 0.37890625, 0.76904296875, -1.23046875, 0.49853515625, -1.7333984375, -1.39453125, -0.0908203125, 0.2315673828125, 0.4423828125, 1.0791015625, 1.3984375, -0.51123046875, 1.8779296875, 0.587890625, 0.058258056640625, 0.2354736328125, -0.36865234375, -0.39501953125, 1.330078125, -0.515625, 0.329833984375, -1.1826171875, -0.32958984375, 0.371337890625, 1.578125, 0.34619140625, 0.09912109375, 0.88427734375, 0.5029296875, -0.75830078125, 0.77392578125, 0.003162384033203125, -0.170166015625, -0.1673583984375, 1.529296875, -0.00627899169921875, -1.904296875, -0.06298828125, 0.234619140625, 0.416015625, -0.77783203125, 2.669921875, 0.318115234375, -0.12353515625, -0.1859130859375, 0.2135009765625, 0.12420654296875, 0.1490478515625, -0.14404296875, -1.173828125, -0.463623046875, -0.80126953125, 1.0703125, 0.58056640625, -1.388671875, 0.0306396484375, -0.41357421875, 0.0810546875, -1.7490234375, 0.59619140625, 0.81982421875, 0.85595703125, -1.4970703125, -0.46484375, 0.93701171875, -0.06854248046875, -0.451416015625, 1.0078125, 1.267578125, -0.1588134765625, -2.02734375, -1.4375, -0.71923828125, 0.1341552734375, 0.85498046875, -0.392822265625, -0.53955078125, 1.5673828125, -0.4716796875, -1.6923828125, 1.3212890625, -0.81494140625, 0.1961669921875, 0.403564453125, 0.7412109375, 0.25439453125, -0.69482421875, -0.422607421875, 0.349853515625, 1.71484375, -0.640625, 1.8037109375, -1.2041015625, -0.64013671875, 0.63330078125, 1.53515625, -0.061309814453125, -0.30615234375, -0.2568359375, 1.5703125, -0.7529296875, -0.408447265625, -0.5986328125, -0.93115234375, -0.104248046875, 0.1624755859375, 0.279296875, 1.6357421875, 1.0625, -1.8427734375, 1.3037109375, 0.8662109375, 1.9833984375, -0.34228515625, 0.305908203125, -0.11737060546875, -0.53759765625, -1.052734375, 0.252685546875, 1.5107421875, -0.86474609375, -2.48828125, -0.11810302734375, -0.5693359375, -2.12890625, -0.7998046875, 0.39697265625, 1.3095703125, 0.71337890625, -1.05859375, -0.177490234375, 0.32470703125, -0.43115234375, -0.25732421875, -1.2080078125, -0.96875, 0.06964111328125, 1.384765625, 0.34375, -0.296875, 0.370361328125, 0.9091796875, -0.08758544921875, 0.6142578125, 0.63623046875, 0.91015625, 0.2198486328125, -1.9208984375, 0.022552490234375, 0.034637451171875, -0.91455078125, -0.301025390625, 0.56494140625, -0.0223541259765625, -0.4658203125, -0.57568359375, -1.0400390625, 0.191162109375, 0.306396484375, 1.7998046875, 1.6005859375, -0.228759765625, -0.1611328125, -0.0036563873291015625, -0.64599609375, 0.268798828125, 0.78955078125, 0.2286376953125, 0.53466796875, 0.98291015625, -1.40234375, -0.6787109375, 0.7392578125, -1.6025390625, -0.90478515625, -2.572265625, -1.2099609375, 1.0439453125, 1.2275390625, 0.22705078125, 0.283935546875, 0.1986083984375, 1.474609375, -0.070068359375, -0.039398193359375, -1.7236328125, -0.91943359375, 0.31689453125, -0.85107421875, -0.1297607421875, 0.94970703125, -2.041015625, 3.271484375, -0.007480621337890625, -0.2122802734375, 2.51171875, -0.82568359375, 0.472412109375, 1.1572265625, -0.1578369140625, -0.469970703125, 0.42724609375, -1.541015625, -0.70849609375, -0.92919921875, 0.2239990234375, -1.0146484375, -0.51904296875, -0.08197021484375, 0.408203125, -0.712890625, 0.7001953125, -0.3212890625, 0.60888671875, 0.6884765625, 0.35986328125, 0.65625, -0.98779296875, -1.564453125, 1.8037109375, 2.080078125, -2.578125, -2.458984375, -0.93994140625, -1.5068359375, -0.029266357421875, 1.009765625, 1.966796875, -0.6943359375, 1.3115234375, 1.244140625, 1.32421875, 1.8681640625, -0.3076171875, 0.61083984375, -0.564453125, 1.3681640625, -1.5458984375, 0.58984375, -0.432861328125, 1.3671875, -0.90234375, 0.422119140625, 0.73974609375, 0.9951171875, -0.8154296875, -0.6279296875, 0.48193359375, 0.21435546875, 0.425048828125, 0.10791015625, 2.625, 0.167724609375, -0.42138671875, -1.041015625, 1.84375, -1.9833984375, -1.3232421875, 0.96240234375, -0.1597900390625, -1.587890625, -0.84423828125, -0.11602783203125, 0.026031494140625, -0.73193359375, 0.1824951171875, -1.58984375, 2.162109375, -1.9228515625, -2.064453125, -0.2012939453125, -0.4619140625, 0.55078125, -1.048828125, 0.822265625, -0.10601806640625, -1.1455078125, 0.96875, -0.314208984375, 0.640625, 0.08056640625, -0.43896484375, -0.7587890625, 1.2822265625, 0.05267333984375, 1.201171875, 0.042633056640625, 0.11395263671875, 0.8212890625, 0.56591796875, -0.603515625, 1.59375, 0.62255859375, -0.1314697265625, -2.162109375, 0.826171875, 1.2431640625, 0.337646484375, -0.06500244140625, -0.30859375, -0.98583984375, 0.04229736328125, 1.2275390625, -1.025390625, 1.453125, 0.10614013671875, -0.61962890625, 0.4072265625, 0.01522064208984375, -0.21728515625, 0.018798828125, 1.802734375, 0.78515625, 2.408203125, -0.7841796875, -0.427978515625, 0.05224609375, 1.5126953125, 0.541015625, 1.5517578125, 0.251708984375, -1.681640625, -2.181640625, -3.33984375, -0.49072265625, -0.352783203125, -1.171875, -1.68359375, -0.74609375, -0.265625, 0.1595458984375, 0.43408203125, -0.8203125, 0.63623046875, 0.2314453125, 0.035247802734375, -0.2144775390625, 1.2958984375, -1.8916015625, 0.86669921875, -0.10400390625, -1.095703125, 0.476806640625, -0.323974609375, -1.59765625, 0.2086181640625, 0.417236328125, -0.2880859375, 0.390625, -0.564453125, -1.4833984375, 0.92822265625, 0.407470703125, 0.84619140625, 1.78125, -0.83740234375, -1.828125, -0.4423828125, 2.619140625, 0.5595703125, -1.34375, -1.1943359375, 0.280517578125, -0.1400146484375, -0.54931640625, -0.89404296875, 1.65625, -0.1407470703125, 2.08984375, -1.19140625, -2.12890625, 0.751953125, -1.255859375, 1.0654296875, -1.947265625, -0.010528564453125, 0.64599609375, 0.469970703125, -0.98046875, -0.1107177734375, -0.48291015625, 1.625, 0.52978515625, 0.494873046875, -0.8154296875, -0.354248046875, -0.20458984375, -0.403564453125, 0.74267578125, -0.11029052734375, 1.1298828125, 0.06927490234375, -0.517578125, 0.74365234375, 0.7119140625, 1.23828125, 0.583984375, 0.92236328125, 0.98876953125, 0.7080078125, -1.107421875, 1.7802734375, -0.728515625, -0.300537109375, -1.3798828125, -1.6875, -0.40380859375, -0.06689453125, 1.2724609375, 0.77294921875, -0.7744140625, -0.0012712478637695312, 0.626953125, -0.44287109375, -1.1025390625, -0.341796875, -1.861328125, 0.42431640625, 1.6640625, 1.13671875, -0.9150390625, 0.10992431640625, -1.2880859375, 1.05859375, -1.619140625, -1.17578125, 1.41796875, -0.7080078125, 0.2413330078125, 0.37451171875, -2.455078125, -1.9619140625, 0.002376556396484375, 0.99609375, 0.2359619140625, 0.034942626953125, 0.81591796875, -0.990234375, 0.210693359375, 0.54443359375, 0.82568359375, 1.0146484375, -0.21337890625, -0.80078125, -0.93212890625, 0.260009765625, -0.99560546875, -0.28564453125, -2.0546875, 1.328125, -0.00327301025390625, -1.814453125, 0.693359375, -0.23095703125, -0.415771484375, 1.43359375, 0.763671875, 1.1279296875, 1.435546875, 0.92822265625, 0.3271484375, -2.2734375, -1.1572265625, 1.6044921875, -3.205078125, -0.89404296875, 0.3935546875, -1.0048828125, 0.81640625, -1.044921875, 0.101318359375, 0.65087890625, -0.6923828125, -1.4921875, -0.436767578125, -0.65869140625, 0.61669921875, 1.142578125, 0.14013671875, 0.41943359375, -1.5498046875, 1.107421875, 0.0709228515625, 0.75390625, 0.27197265625, 1.78515625, -0.8466796875, 0.75830078125, 1.376953125, -0.64306640625, -0.27099609375, 1.5732421875, 0.2091064453125, 0.32568359375, -0.95361328125, 0.44677734375, -1.18359375, 1.0341796875, -0.307861328125, 0.5546875, 1.080078125, -0.30078125, 0.2147216796875, 1.7724609375, -1.056640625, 2.193359375, 0.443359375, -1.7763671875, -0.9765625, -1.9580078125, -0.55126953125, -0.1602783203125, 1.041015625, 0.6708984375, -0.1396484375, -0.95556640625, 0.43359375, -1.6259765625, 0.470458984375, 2.33203125, 2.361328125, 0.01319122314453125, 2.6015625, 0.07806396484375, -0.209716796875, 0.56982421875, 2.771484375, -0.63916015625, -0.81689453125, -0.004833221435546875, -1.0185546875, 0.2373046875, -1.8095703125, -0.9033203125, 0.051300048828125, 0.414794921875, 0.8037109375, -1.1181640625, -0.471923828125, 0.9560546875, 0.299560546875, 0.11419677734375, 1.5400390625, 1.353515625, -0.66357421875, 1.12109375, 0.369873046875, 1.4423828125, -0.60205078125, -1.3095703125, 1.7119140625, 0.1583251953125, -0.068115234375, 0.16357421875, 0.9560546875, -1.0166015625, 0.2130126953125, -1.3427734375, 1.1875, -1.1328125, -0.63720703125, -1.064453125, 2.58984375, -0.1461181640625, -0.23046875, -0.260986328125, 1.2275390625, 0.1676025390625, -0.40625, 1.87109375, 0.1107177734375, 1.3916015625, -0.60205078125, 0.0750732421875, 0.377197265625, -0.32373046875, -2.2109375, 0.880859375, -1.2314453125, 1.0869140625, -1.0947265625, -0.34130859375, -0.80712890625, 0.75439453125, 0.48876953125, -2.392578125, 0.3857421875, 0.97509765625, -0.4658203125, -0.70654296875, 0.020538330078125, 0.50146484375, -0.64599609375, 1.1103515625, -0.1593017578125, -2.404296875, -0.2015380859375, -1.75390625, 1.236328125, 0.6982421875, -1.908203125, 1.568359375, -0.061431884765625, -1.6728515625, -0.953125, -0.6123046875, 1.6181640625, -0.37060546875, -0.966796875, 0.50341796875, 0.5791015625, -0.052947998046875, 2.251953125, 0.07061767578125, -1.6142578125, -0.00066375732421875, 0.151123046875, 0.75048828125, 0.1331787109375, 0.44970703125, 0.2496337890625, 0.494873046875, -0.2626953125, -0.6875, -1.08984375, 1.634765625, -1.0087890625, 1.263671875, 1.0966796875, 0.1292724609375, 0.322509765625, -1.345703125, -0.2283935546875, 0.037261962890625, -0.775390625, 0.912109375, 2.5234375, 1.109375, 0.12030029296875, 0.14208984375, -0.60205078125, -0.314453125, -0.26708984375, 0.1600341796875, -0.7080078125, -1.861328125, -0.75830078125, 1.751953125, 0.074951171875, -0.20068359375, -0.354736328125, 0.8037109375, -0.053070068359375, 1.162109375, -0.740234375, -0.264404296875, -1.083984375, -0.301025390625, 0.133056640625, -1.3994140625, 0.9306640625, -0.1934814453125, -0.9697265625, -1.197265625, -0.19775390625, 1.03125, 0.814453125, -1.1376953125, 0.2159423828125, -0.90869140625, 0.75830078125, -0.79443359375, -0.65966796875, 0.53369140625, 0.35888671875, -0.884765625, 1.59765625, -1.2822265625, 1.5498046875, -1.7021484375, -0.455322265625, -1.169921875, 0.0261077880859375, 0.416259765625, 0.70263671875, -0.3681640625, 1.6669921875, 1.3134765625, -0.89697265625, 0.69970703125, -1.197265625, 0.452880859375, 1.1640625, 0.2412109375, 0.454833984375, -0.662109375, -1.3193359375, 1.724609375, -0.85400390625, -1.4755859375, 0.7978515625, -0.08453369140625, 0.028900146484375, -0.58642578125, 0.355224609375, -0.2318115234375, -1.01953125, 0.0287017822265625, 1.6669921875, -0.028656005859375, 0.7568359375, 1.47265625, 0.31640625, 1.6025390625, -0.7255859375, 0.22509765625, -0.490234375, -0.12841796875, -0.4833984375, -1.2080078125, 0.6396484375, -1.34375, -0.2342529296875, 0.061309814453125, 2.310546875, -2.982421875, 0.87255859375, -1.19921875, -0.36572265625, -0.029205322265625, -0.1522216796875, 0.102783203125, -1.4111328125, -0.07354736328125, -0.8583984375, -0.0711669921875, -0.367919921875, -0.06475830078125, -1.0576171875, -0.68017578125, -0.31689453125, 1.267578125, 0.014373779296875, -1.7041015625, 1.8310546875, 2.015625, 2.830078125, -0.1826171875, 0.50146484375, -0.251220703125, -0.1412353515625, 0.71484375, -0.463134765625, -0.4892578125, 1.9873046875, 0.10577392578125, -0.1563720703125, -1.15234375, -0.8701171875, 0.87548828125, -0.40380859375, 1.0595703125, -0.62548828125, -0.87353515625, 0.230712890625, -0.61328125, 0.257080078125, 2.296875, 1.224609375, 0.06268310546875, 2.037109375, -0.74560546875, 0.98388671875, -0.0599365234375, -0.51611328125, -0.66748046875, 0.51318359375, -1.404296875, -0.3427734375, -0.22412109375, -0.81787109375, 1.251953125, 1.4755859375, -0.51318359375, 0.99755859375, 0.1981201171875, 1.3349609375, 0.03131103515625, -0.408935546875, 1.822265625, 0.81201171875, 0.2469482421875, 1.0517578125, -0.72900390625, 0.2900390625, 0.548828125, 0.8984375, -0.301513671875, 0.369140625, -1.4599609375, -0.0433349609375, -1.3837890625, -0.54638671875, -1.201171875, -0.875, 1.826171875, -0.74609375, -1.66796875, -1.0322265625, 0.9931640625, -0.297119140625, -0.95849609375, 1.220703125, -0.9326171875, 0.70654296875, 1.1591796875, -0.0628662109375, -0.1495361328125, 0.016204833984375, 1.208984375, 0.400390625, -0.461669921875, -0.91357421875, 0.68896484375, 0.0115203857421875, -0.94189453125, -1.8837890625, 1.2490234375, -0.2724609375, 0.61865234375, 1.373046875, -0.5751953125, 1.80078125, 1.1513671875, -0.09307861328125, 0.73388671875, -1.4150390625, 0.8974609375, 1.8193359375, -0.3916015625, -0.399658203125, 1.77734375, 1.056640625, -1.0947265625, -0.41943359375, 1.171875, 0.07696533203125, 0.67333984375, 0.0753173828125, 2.578125, 0.1424560546875, -0.9853515625, 0.87890625, 0.28955078125, 2.31640625, 0.67138671875, 1.0390625, -0.78857421875, -0.1158447265625, 1.169921875, 1.78515625, 0.2115478515625, 1.255859375, 0.050750732421875, -1.9111328125, 0.1202392578125, 1.904296875, -1.2431640625, -0.282958984375, 1.62890625, -1.6484375, -2.375, 0.1153564453125, -0.88134765625, -0.4677734375, 1.150390625, -1.0400390625, -1.9794921875, -0.5771484375, -1.005859375, 0.048309326171875, 0.63037109375, 0.85302734375, 0.1038818359375, 0.54736328125, -1.041015625, 0.353271484375, -0.303466796875, 0.280029296875, 0.06463623046875, 0.5439453125, 0.890625, 0.374755859375, 0.391357421875, -0.7216796875, 0.509765625, -0.0848388671875, 1.1533203125, 0.52099609375, -0.9365234375, 0.258544921875, 0.5263671875, -0.090087890625, 0.67138671875, -0.026214599609375, -0.358642578125, -0.326171875, 1.0498046875, 0.87353515625, 0.4423828125, 0.425537109375, -1.27734375, 1.0234375, -0.23193359375, 0.238525390625, 0.65966796875, -0.708984375, -0.395751953125, -0.036468505859375, 0.08148193359375, 0.06787109375, 1.5087890625, 0.32373046875, 0.35595703125, 1.09375, -1.7666015625, 1.3388671875, 0.328369140625, -0.91845703125, 1.0, 0.78271484375, -0.10284423828125, -2.541015625, -0.312744140625, 1.3701171875, -0.187744140625, 0.11614990234375, 1.7451171875, 0.98291015625, -1.060546875, 0.6865234375, 0.321044921875, 1.3330078125, 0.11328125, 1.228515625, 0.38525390625, -0.775390625, -0.282958984375, 0.65380859375, -0.0241546630859375, -1.2021484375, -0.0924072265625, -1.50390625, 1.8115234375, 1.94921875, 0.74755859375, -1.8232421875, -0.28076171875, 0.9833984375, -0.0777587890625, 1.4736328125, -0.9951171875, -1.14453125, 0.82373046875, -0.31298828125, 0.072998046875, 2.05859375, -1.736328125, 0.10845947265625, 0.82080078125, 0.79736328125, 0.80712890625, -1.759765625, 1.0673828125, -0.15185546875, 1.01953125, 0.28173828125, 0.50537109375, 0.52294921875, 0.93505859375, -1.0146484375, 1.1337890625, -0.51220703125, 0.314697265625, 0.323486328125, -0.489013671875, -0.4462890625, 0.1986083984375, 0.10467529296875, 0.72314453125, -1.453125, -0.70068359375, 1.9638671875, -0.63818359375, -0.371337890625, 0.83203125, 0.6552734375, 0.57958984375, 0.0648193359375, 0.1942138671875, 0.3193359375, 0.955078125, 0.6669921875, -0.58740234375, 0.36962890625, -0.2261962890625, -0.1622314453125, -0.42578125, -1.2431640625, 1.755859375, 0.34912109375, 0.76708984375, -0.09161376953125, -0.1455078125, 1.494140625, 0.986328125, 2.525390625, 0.56103515625, 0.802734375, -0.1055908203125, 1.0654296875, -0.716796875, -0.73193359375, 0.260498046875, 1.9228515625, 0.3115234375, -1.3212890625, -1.0830078125, 0.066162109375, -0.18505859375, 2.19140625, 0.75634765625, 0.70654296875, 0.0178680419921875, 2.34375, -0.875, -0.343505859375, -0.580078125, -0.192626953125, 0.93994140625, 0.11474609375, -0.92529296875, -0.343994140625, 0.53662109375, 1.1806640625, -0.99609375, -0.374755859375, 0.69775390625, 1.599609375, 1.7685546875, 1.3994140625, -0.0004944801330566406, -0.439208984375, 0.391357421875, 1.62890625, -1.1142578125, 1.5087890625, -1.5068359375, -1.287109375, 0.75732421875, 0.84619140625, 1.7177734375, 0.416015625, 0.97119140625, -0.79833984375, -0.81787109375, 0.003955841064453125, -0.60595703125, -1.935546875, 1.4755859375, 0.01215362548828125, 0.387939453125, 0.347900390625, -0.374755859375, 0.533203125, 0.425537109375, -0.64892578125, -0.0460205078125, 0.005741119384765625, 0.2366943359375, -0.96923828125, -0.177734375, -0.5458984375, 1.2412109375, 1.0947265625, 1.0654296875, 1.5634765625, -0.8408203125, 0.22509765625, -0.76806640625, 0.1478271484375, 1.298828125, 0.65771484375, 0.78369140625, -1.1376953125, -1.765625, 0.398193359375, 0.22412109375, -0.497802734375, -1.0966796875, 0.98828125, -1.0927734375, 0.15576171875, -0.52880859375, 1.2412109375, -0.84423828125, 0.55419921875, -0.2088623046875, -0.51611328125, -0.52880859375, -1.810546875, -0.2900390625, 0.484375, -1.2705078125, -1.017578125, 0.218505859375, -0.73486328125, -1.359375, -0.7158203125, 0.1322021484375, 1.3671875, -1.234375, 0.140380859375, -1.099609375, 0.6005859375, 2.16015625, 0.71728515625, -0.459228515625, -1.060546875, -1.0556640625, -1.9638671875, 0.59228515625, 1.1943359375, -0.08807373046875, -0.5673828125, 0.10040283203125, -0.73193359375, -0.80615234375, -0.54541015625, -1.2822265625, 1.994140625, -1.1181640625, 2.1796875, 0.080078125, -2.89453125, 0.26611328125, 0.9462890625, 0.24072265625, -0.2359619140625, -0.66162109375, 0.82861328125, -0.7626953125, -0.424072265625, 0.02410888671875, -0.23388671875, 0.052825927734375, 1.0859375, 0.455810546875, -0.01287078857421875, 0.2364501953125, 0.490478515625, -1.1240234375, -0.6025390625, -1.7431640625, -0.79345703125, -0.52685546875, 0.9453125, -0.4765625, -0.2138671875, -1.05078125, 0.52783203125, -0.4951171875, -1.375, -0.319091796875, 0.451904296875, 0.243408203125, -0.01123809814453125, -0.951171875, 0.9404296875, 0.073486328125, -0.35888671875, -0.1744384765625, -0.349365234375, -0.8896484375, 2.900390625, -1.59375, 0.26904296875, -0.8818359375, 0.292724609375, 1.7646484375, -0.72412109375, 1.9970703125, 0.12103271484375, 0.50439453125, -0.83203125, -2.283203125, -1.0556640625, 0.62841796875, -0.53662109375, 1.5302734375, 1.2880859375, 0.225830078125, 2.341796875, -1.2900390625, -1.7119140625, -1.19921875, 1.0126953125, 0.74951171875, 0.6591796875, 0.2060546875, -0.43994140625, -1.1357421875, -0.73974609375, 0.556640625, 0.2431640625, 0.611328125, 0.315673828125, -0.464111328125, 0.347412109375, 0.41064453125, 1.599609375, 0.05303955078125, 0.552734375, 0.919921875, 0.681640625, -0.34521484375, -1.49609375, 1.1435546875, -0.99365234375, -0.257080078125, 1.2099609375, 0.29736328125, -0.1937255859375, 1.1455078125, 0.09912109375, 2.046875, 0.10760498046875, -0.89697265625, -1.95703125, -0.1383056640625, 0.69580078125, 0.26611328125, -1.5654296875, -0.484619140625, -1.5576171875, 1.15625, -0.1156005859375, -1.076171875, -0.270263671875, -0.41748046875, 1.3671875, 0.270263671875, -1.0009765625, 0.91796875, -1.0478515625, 1.416015625, 0.386474609375, -1.0947265625, -2.798828125, 0.327880859375, 0.096435546875, 0.85498046875, -0.123779296875, -1.24609375, 0.1412353515625, 0.7802734375, 1.263671875, 0.69970703125, 0.8798828125, 1.5634765625, 1.3603515625, 1.2021484375, 1.408203125, -1.3818359375, -2.64453125, -2.310546875, 0.859375, 1.6025390625, 1.54296875, 0.376220703125, 0.11566162109375, -0.091552734375, -2.46875, -2.048828125, -0.6943359375, 0.91845703125, -0.8193359375, 0.410400390625, -0.316162109375, 0.74462890625, 2.392578125, 1.1845703125, 0.447998046875, -0.026214599609375, 0.04864501953125, 0.69970703125, -0.486572265625, -2.19921875, -0.01116943359375, 0.8359375, 0.59130859375, -0.9462890625, -0.693359375, -0.247802734375, -2.638671875, 0.44140625, -0.291015625, 0.8798828125, -1.748046875, 1.72265625, 0.03448486328125, -0.34619140625, -0.85791015625, 0.037384033203125, 1.0859375, 0.89794921875, 1.171875, -0.022125244140625, -0.2137451171875, -0.7158203125, 0.6943359375, 1.09375, 2.1484375, -0.08001708984375, 0.52490234375, -0.2396240234375, -0.60986328125, -0.5908203125, 0.48828125, 0.87060546875, -0.7392578125, 0.07598876953125, 0.05450439453125, -0.1273193359375, 1.45703125, -1.767578125, -1.767578125, -0.73779296875, 0.85205078125, 0.96484375, 0.319091796875, -0.302734375, 1.3486328125, -1.36328125, 0.427734375, 1.6376953125, -0.5234375, -0.442626953125, 1.0244140625, -0.72216796875, 1.271484375, 0.28125, -0.80615234375, 0.08648681640625, 0.352783203125, 0.53173828125, -0.73095703125, -0.31396484375, 0.259033203125, 0.46728515625, 1.076171875, -0.324462890625, -0.1925048828125, 0.1605224609375, 0.1488037109375, 0.178955078125, 0.60400390625, 1.4609375, 1.2607421875, -0.6455078125, -0.67626953125, -0.7880859375, -0.40576171875, -1.009765625, 0.65185546875, -0.669921875, 1.478515625, -0.66455078125, -0.1043701171875, -0.916015625, 0.71630859375, -0.28076171875, 0.68798828125, 1.712890625, -1.5439453125, 0.254150390625, -0.59521484375, -1.3515625, 1.3583984375, -0.29443359375, 1.85546875, -0.70166015625, 0.61474609375, 0.2108154296875, 0.96240234375, -0.1005859375, 0.52197265625, 0.07769775390625, -0.51953125, 0.97705078125, 1.8564453125, 1.2666015625, 0.740234375, 0.054962158203125, 0.295166015625, -0.7060546875, -0.90771484375, 0.73046875, -1.0654296875, 0.137451171875, 0.8017578125, 0.8701171875, -0.5546875, 1.6064453125, -0.2108154296875, -1.4716796875, -0.873046875, -0.3271484375, 0.84765625, 0.5263671875, -2.28125, -0.75537109375, 0.278564453125, 0.2020263671875, 0.95703125, -0.292236328125, 1.970703125, 0.3115234375, 1.4970703125, 1.2939453125, 1.2626953125, -0.2164306640625, -0.278076171875, 0.130126953125, -0.1888427734375, -0.35107421875, -0.70751953125, -0.85400390625, 1.6826171875, 0.91748046875, 0.444580078125, -0.58251953125, -1.232421875, -1.134765625, 1.63671875, -0.1456298828125, 0.258056640625, -0.1630859375, 0.419677734375, -0.34814453125, 0.46484375, -0.018096923828125, 0.62255859375, 2.462890625, -0.29638671875, -0.3515625, 1.4755859375, -0.0457763671875, 1.244140625, -0.460205078125, 0.783203125, 0.47900390625, 0.1427001953125, 0.1123046875, -0.93701171875, 0.45654296875, 0.6748046875, 0.45068359375, 0.48974609375, 0.034149169921875, 0.0718994140625, -1.1123046875, 0.5478515625, -0.34375, -0.07891845703125, 0.15771484375, -1.763671875, -0.59912109375, -1.3916015625, -0.9765625, 0.89794921875, -0.5234375, -0.03717041015625, 0.0240325927734375, 1.6318359375, -0.434326171875, 1.5673828125, 0.362548828125, 0.69482421875, 0.1531982421875, -1.033203125, -0.98046875, -0.1234130859375, 1.958984375, -0.91162109375, 1.2236328125, -0.442138671875, -0.09716796875, 2.095703125, 1.396484375, 1.275390625, 0.79248046875, -0.2236328125, -0.708984375, 0.08026123046875, 1.9482421875, -0.38232421875, -0.77734375, -0.14453125, 0.6474609375, -0.955078125, 0.50634765625, -1.0087890625, 0.8623046875, 0.51220703125, 0.9873046875, 0.25439453125, 0.12890625, -0.152587890625, -0.78662109375, 1.6513671875, -1.2802734375, -0.38720703125, -2.138671875, -0.447265625, -0.4326171875, -1.1962890625, 0.2315673828125, -1.37890625, -0.77734375, 1.626953125, -0.28173828125, 0.705078125, 0.01424407958984375, 0.48046875, 0.9736328125, -0.207763671875, 1.2333984375, -0.5908203125]], "dtype": "float16"}, 1], "e": {"__nd__": [-916.6903686523438], "dtype": "float32"}}, {"a": [{"__nd__": [[7.5]], "dtype": "float32"}, 0], "e": {"__nd__": [7.5], "dtype": "float32"}}, {"a": [{"__nd__": [[116.1875, -28.5625, 44.3125, -128.75, 174.75, -87.9375, -40.78125, 48.71875, -46.0, -182.875, 124.8125, -7.1484375, -10.7578125, 157.375, -86.75, 210.375, -18.84375], [-97.375, 3.6953125, -40.5, -34.34375, 166.0, -17.65625, -1.9541015625, 127.375, 63.0, -99.875, -51.46875, -27.96875, 28.75, 49.96875, 88.0, 174.375, -77.5625], [131.125, 42.78125, -161.375, 31.015625, -19.609375, -97.25, -137.875, -27.296875, 129.875, 55.71875, 69.8125, 218.875, 128.25, -17.625, -131.625, -140.0, 11.71875], [122.9375, -156.5, -59.625, 204.0, 98.75, 54.65625, 174.625, -211.25, -35.96875, -14.4140625, -5.67578125, -166.375, 145.875, 159.125, 205.25, -148.75, 77.75], [-22.3125, -146.875, -51.3125, -11.0234375, -145.875, -173.0, -58.09375, -43.3125, -123.875, 111.125, -26.078125, -23.984375, -56.0, -27.859375, 85.0, 122.4375, -85.75], [51.28125, 79.0625, 43.15625, 55.03125, -96.8125, -114.0, 7.77734375, -104.9375, -129.0, 75.0625, -61.84375, -33.71875, 38.03125, 198.5, 30.796875, 1.6953125, -26.953125], [-24.90625, -10.4296875, -39.5625, 24.546875, -6.59765625, 66.8125, 113.375, -26.46875, 128.375, 37.625, 75.3125, -156.625, -42.21875, -79.5625, -123.4375, 40.96875, -53.03125], [-42.1875, -78.625, 88.3125, -28.921875, 27.0, -43.21875, 114.25, 157.875, 2.75390625, -42.4375, -30.203125, -44.59375, 50.96875, -123.8125, 92.8125, -108.4375, 86.3125], [-120.25, -61.71875, -86.1875, -16.421875, 50.25, 18.328125, -11.3203125, 40.78125, 146.625, -101.0, -216.0, -109.375, -48.03125, -118.875, 42.1875, 82.25, 65.9375], [83.75, -170.25, 93.5, -38.40625, -108.625, -58.46875, -96.5625, -26.734375, 98.625, 91.375, -36.09375, 132.625, 118.125, -34.46875, -208.375, -42.40625, -135.0], [-12.8828125, 54.59375, -34.34375, -92.5, 38.9375, -9.40625, 106.5625, -60.9375, -36.375, 106.875, 72.8125, 23.359375, 207.625, 132.5, 145.875, 30.90625, -45.53125], [-73.25, 18.34375, 112.0, -120.9375, -91.4375, 156.875, 17.78125, -30.203125, 79.4375, 140.625, 188.25, -82.875, -35.03125, -108.125, 97.6875, -109.6875, -2.75], [240.875, -42.0, 44.4375, -137.75, 218.5, 130.25, 40.5, -39.15625, -73.0, -68.4375, 42.28125, -151.875, 82.125, -50.625, 47.3125, 168.125, 166.375], [-85.6875, -142.0, 121.0, 7.734375, 92.5, 1.916015625, 240.375, -208.5, 96.4375, -20.265625, -74.0625, -138.25, -37.0625, 97.4375, -93.3125, -159.625, 162.75], [-74.9375, 14.65625, -15.90625, -140.5, 139.5, -165.25, -21.796875, -37.5, 26.3125, -23.109375, 298.5, 202.5, 179.875, -47.71875, -70.5625, 116.6875, -66.9375], [85.5, 93.3125, 28.625, -81.9375, -85.1875, -132.375, -36.5625, 3.60546875, -14.6015625, 38.78125, -13.1484375, 134.75, 35.90625, -21.796875, -11.671875, -86.8125, -70.125], [-248.875, -97.125, 50.3125, 155.0, 53.125, -83.5625, 105.9375, -41.59375, -41.5, -32.0625, 56.1875, -175.625, -145.875, -117.25, 0.822265625, -11.1796875, -32.84375], [33.8125, -234.0, -33.1875, -50.71875, -44.28125, -84.125, -0.421875, -270.25, 121.875, 19.65625, -96.0625, -12.890625, 54.03125, -54.25, 97.125, 64.9375, 56.625], [112.375, -177.25, -49.5, -6.67578125, -160.25, -120.875, 49.5, 80.9375, -40.46875, -30.796875, 40.53125, 60.375, 105.25, -5.59375, -52.59375, -110.8125, 110.125], [-14.8515625, 115.75, 31.875, 78.5, 17.765625, 54.40625, -23.765625, 19.375, 128.125, -39.375, 31.59375, 56.21875, 31.21875, 69.3125, -94.5625, 39.53125, 59.34375], [154.625, 152.875, 11.1171875, -145.0, 81.3125, -166.875, -88.8125, 22.765625, 47.1875, 3.595703125, 46.875, 0.78076171875, 42.0625, -17.84375, -27.046875, -106.625, -81.25], [-61.71875, -76.0625, 172.5, -166.625, 276.0, 170.75, 85.25, -166.375, -24.5, -42.21875, 47.375, -32.59375, 138.375, 11.15625, -44.09375, 71.3125, -3.94140625], [49.15625, -2.314453125, -146.0, -24.75, 50.46875, 133.125, -67.75, 10.578125, 113.625, 49.46875, -170.125, -39.78125, -55.875, -137.625, -40.25, 19.78125, -13.9765625], [138.125, -71.875, -64.4375, 10.640625, -80.125, 148.25, -134.5, -102.875, -33.53125, -17.515625, -48.4375, -139.75, -27.28125, -10.9296875, -15.328125, -20.75, 152.625], [-27.78125, -49.34375, -74.875, 129.625, -81.125, 25.65625, -38.40625, -38.34375, -24.984375, 13.6015625, -81.625, 58.90625, -74.125, -224.75, -51.375, -165.5, -130.375], [-28.734375, 105.375, 11.8125, -24.953125, -8.53125, 143.125, -114.3125, -73.25, 105.25, -19.21875, 218.75, -95.375, -117.375, 92.3125, -70.4375, -46.0, 75.375], [-49.46875, -39.40625, -88.875, 157.5, -165.125, 104.25, -73.9375, -161.125, 93.375, 127.0, -107.75, -27.78125, 62.875, -43.4375, 147.5, -47.375, 133.625], [-71.6875, -94.3125, -119.6875, 136.125, 28.453125, 81.5625, 16.109375, 14.1953125, -88.3125, 9.8515625, -108.375, 74.75, -91.5625, -12.8203125, 99.4375, -9.1484375, -65.375], [-63.96875, -136.625, 26.109375, -114.875, -146.875, -65.1875, -65.5625, 47.78125, -34.71875, 25.921875, -66.75, -24.390625, 62.8125, 81.5, 56.53125, -7.84375, 130.75], [54.9375, 56.15625, -66.375, -69.5, -94.5625, -70.125, 19.25, 102.6875, 37.09375, 23.6875, -27.203125, -0.11590576171875, 134.25, 37.4375, -29.46875, 39.03125, 159.0], [-79.9375, 30.546875, 153.0, -21.390625, -42.75, -127.5, 24.984375, 56.59375, -72.0625, 153.75, -90.125, 3.740234375, 77.25, 214.375, 135.125, 71.1875, -24.0], [21.859375, 3.083984375, -18.203125, 123.8125, 100.8125, 150.125, -45.8125, -125.75, 17.46875, -182.125, -161.5, 168.625, 143.0, -213.25, 41.65625, -0.5009765625, -49.53125], [3.142578125, -75.4375, 52.40625, -9.3046875, -79.8125, -53.1875, -80.875, 130.75, -91.3125, 6.52734375, -45.46875, -9.8984375, 122.3125, -92.375, -36.75, -158.875, 248.75], [136.25, 148.125, 128.75, 5.5078125, -116.75, -147.875, 119.9375, -177.875, -25.453125, -55.375, -52.25, 106.8125, -34.40625, -9.8828125, -55.3125, 248.625, 44.71875], [-129.5, -100.3125, -25.5625, 104.3125, -29.953125, 57.6875, -207.0, 28.9375, -54.34375, 104.6875, 232.375, 34.28125, -104.8125, 109.375, 97.875, -62.375, -92.5], [9.40625, -19.40625, 33.40625, 32.65625, -77.875, -91.75, 7.953125, -123.75, 208.375, -38.71875, -62.0, 73.25, 37.75, 3.9921875, 151.125, 17.34375, 91.375], [87.5625, 32.4375, -16.921875, 207.5, -71.0625, -8.9453125, -91.9375, 217.0, 71.3125, 141.75, -27.8125, -8.375, -159.0, -254.875, 37.40625, 73.8125, 63.0], [83.6875, -116.0, -198.0, -313.25, -164.25, 99.75, -17.3125, -269.25, -91.625, -20.875, -24.875, -67.9375, 38.84375, -76.3125, 78.25, -13.6171875, -161.25], [-60.1875, 195.25, -120.4375, 90.75, -42.96875, -36.46875, -91.125, -146.25, -20.296875, 87.1875, -194.75, -48.5625, -51.34375, -30.109375, -9.9609375, -64.0, -83.625], [-38.875, 43.0, -199.5, 22.78125, -130.25, -225.5, 132.25, -101.25, -105.375, 54.96875, -80.0, -7.2734375, -56.8125, -11.765625, 15.5078125, -176.0, 64.5], [-44.4375, -89.25, 66.125, 104.25, -70.25, 137.625, -20.9375, -20.5625, 183.375, -124.0625, 24.8125, 88.75, -27.875, 8.6484375, -71.8125, -74.3125, -16.609375], [114.875, 11.265625, 27.640625, -53.28125, 78.5, 20.09375, 20.34375, -80.9375, -14.1953125, -33.15625, 114.0, 19.359375, -79.875, 177.875, -42.875, 225.75, 97.625], [9.359375, -86.25, -2.384765625, 57.9375, -60.84375, -27.15625, -153.5, 116.0, 3.0, -259.75, 137.375, -129.125, -0.280517578125, 102.6875, -47.96875, 91.3125, -79.9375], [154.75, -25.890625, 24.265625, -73.6875, -36.625, 106.3125, -1.541015625, -57.78125, 47.9375, 103.9375, 161.125, -171.875, 161.375, 148.125, 182.5, -107.875, -117.6875], [-61.40625, -41.84375, -43.40625, 185.75, 15.5859375, -189.75, -35.46875, -170.625, -71.875, 22.484375, -26.421875, 52.875, 75.375, 109.1875, -52.6875, 76.0, 16.765625], [22.484375, -138.0, 137.75, 29.875, 117.1875, 80.6875, 284.5, -261.0, 192.5, -70.4375, -94.1875, 14.8515625, 303.0, 3.087890625, 67.4375, 44.03125, 57.40625], [-130.375, 224.5, -27.515625, 107.5, -107.9375, 99.9375, 66.875, 100.5, -47.59375, 121.875, 61.0625, 17.09375, -85.5, 79.4375, -44.9375, 32.375, 158.25], [123.5625, 9.4921875, -42.03125, 60.625, 51.375, 98.5625, 74.8125, 114.8125, 71.0, 2.626953125, -78.125, -2.232421875, 54.9375, -5.234375, 8.8203125, 67.5, 37.28125], [26.890625, -24.21875, 31.640625, 101.5, 18.640625, 224.125, 20.21875, 165.375, 84.125, 23.21875, -31.15625, 4.3515625, 11.6640625, 21.984375, -46.375, -55.3125, 14.078125], [-68.125, 81.5, -106.625, -59.40625, -158.875, 129.25, 32.28125, 209.625, 86.0, 3.40234375, -77.1875, 55.1875, 7.9375, -34.0625, 95.75, 111.375, -165.625], [-12.3125, -63.6875, -74.4375, 55.65625, 74.1875, -112.4375, 53.40625, 116.25, -42.84375, 45.40625, -34.09375, 19.90625, 116.8125, 44.40625, 82.1875, -49.75, -56.28125], [162.5, 86.3125, -91.625, 40.8125, 93.8125, -88.6875, -73.6875, 169.5, -86.0, 16.609375, 43.96875, -125.75, 92.1875, -88.0625, -25.296875, 18.390625, -132.25], [-37.34375, -107.8125, 98.375, -22.21875, -17.375, 97.8125, -48.9375, -13.625, 14.203125, 59.1875, -14.484375, 27.765625, -26.546875, 5.62890625, 32.0, -106.25, 31.21875], [101.125, 21.890625, -71.5625, -109.25, 25.421875, 175.375, 107.9375, -32.84375, 67.3125, 146.25, -31.671875, -50.96875, -68.0, 79.6875, -111.25, 27.515625, -121.625], [-28.84375, -28.671875, -23.234375, 65.0625, 29.515625, 15.890625, 117.375, 43.03125, 18.625, 27.515625, 51.1875, -69.5625, 16.5625, 52.8125, 160.25, 98.5625, 8.4765625], [67.9375, 28.03125, 9.5078125, -23.109375, -82.875, -37.375, 21.6875, -8.5546875, -15.59375, -180.125, 20.875, -52.71875, -48.90625, -89.375, -7.29296875, -108.1875, -6.640625], [-121.9375, -28.4375, -30.6875, 173.125, 68.125, 152.125, 67.375, -56.03125, -89.0, -23.015625, 84.4375, 69.0, 110.25, 29.140625, -5.015625, 174.875, 153.125], [-139.375, -156.625, -38.59375, -186.5, 7.07421875, -152.5, 79.625, -57.71875, -120.375, 43.09375, -105.4375, 39.53125, 46.59375, -93.25, -16.015625, -73.25, 144.0], [106.75, -59.09375, -12.0625, -35.3125, -67.625, -33.46875, 105.25, 50.9375, 71.75, 18.765625, 14.953125, -148.75, 16.015625, 121.3125, 55.25, 10.765625, -127.125], [-69.125, -64.25, -31.53125, 52.96875, 91.4375, -162.25, 80.75, -44.84375, 156.25, -110.625, 23.6875, 31.28125, 97.8125, -121.5625, -204.5, 64.25, 7.53515625], [144.375, -153.5, -157.0, -5.828125, 91.0, 48.9375, -87.125, 117.4375, -55.40625, -147.375, 53.4375, 113.9375, -65.1875, -205.375, 255.25, -94.125, 178.375], [49.6875, -115.125, -90.25, 170.125, 115.0625, -37.25, 13.71875, 122.3125, 61.0, -68.6875, 87.5, 108.875, 185.875, -93.0, 43.1875, -65.625, -49.84375], [93.9375, -169.75, 23.34375, -118.875, 52.625, -67.3125, -98.8125, 1.6181640625, 101.5625, -62.75, 23.421875, -132.625, 24.875, -74.125, -45.28125, -40.875, 14.0703125], [-35.28125, 6.12109375, 36.59375, -165.875, 181.625, -23.859375, 30.296875, 70.3125, 22.59375, -59.5, 130.25, 222.875, 44.84375, 9.40625, -12.75, 255.375, -75.25], [-35.6875, 88.625, -38.5625, 142.5, 8.078125, 94.75, 114.4375, -22.890625, -34.84375, -81.5, -137.625, 79.125, -85.4375, 353.25, -131.375, -165.25, 1.853515625], [-139.875, -10.0, 95.8125, 82.5625, -110.1875, 105.9375, 18.890625, 55.28125, -77.8125, 7.2890625, 11.453125, 35.78125, -8.265625, 38.21875, 90.8125, -20.984375, 112.0], [-118.3125, 123.9375, -39.1875, -50.625, -49.125, -220.625, -67.4375, 127.8125, 58.5, -34.34375, 18.625, -243.5, 60.75, 56.6875, 51.375, -107.8125, 121.1875], [-8.8671875, -39.53125, -49.375, 32.78125, 221.125, -88.375, 26.265625, 64.75, -52.25, -57.53125, 93.375, 86.25, 82.625, 103.0625, 45.8125, 98.8125, 87.5], [-83.3125, 129.125, 145.375, -49.28125, -146.625, -115.5625, -1.0556640625, -15.8984375, -4.23046875, -169.625, 119.6875, -17.0625, -2.3203125, -68.5, -0.89794921875, 154.25, 316.25], [-85.4375, 192.375, -36.15625, 170.875, -39.5625, 205.25, 113.1875, -111.8125, 77.9375, 130.125, -216.75, 34.125, -15.0859375, -70.25, 18.953125, -32.96875, 56.59375], [104.3125, 11.4609375, -180.875, 61.90625, -178.75, 11.4140625, 101.75, 152.125, -15.9375, -70.9375, 46.53125, 47.84375, -76.625, 101.3125, 43.34375, -36.0625, -13.453125], [-36.375, 237.875, -96.25, -207.875, 68.4375, 142.25, -12.0078125, 75.9375, 41.0, 14.0234375, -40.1875, -132.625, 260.75, 134.625, 1.3671875, -142.5, -116.9375], [-100.0625, 7.15234375, 104.75, -0.7705078125, 256.5, 4.91796875, -25.015625, 124.0, -118.0, 87.0625, 142.625, -9.328125, -150.375, 169.75, 201.875, -92.0625, 31.640625], [49.9375, -66.625, 108.0625, 127.625, 59.5625, 147.5, 52.15625, 122.5, -61.8125, 21.828125, -89.25, 12.171875, -27.984375, 87.9375, 145.375, 80.875, 29.046875], [-71.5625, 27.15625, 113.375, 63.40625, -68.4375, -58.1875, 11.9765625, 58.78125, 133.25, 45.75, -74.6875, -5.16796875, 39.15625, 123.3125, 72.8125, 77.3125, 214.0], [-32.0625, 46.375, 72.9375, -15.734375, 134.75, 13.1328125, -2.091796875, 176.875, -58.375, 132.375, 3.873046875, -24.109375, 57.90625, -2.787109375, 86.75, 38.625, -70.0625], [77.8125, -32.625, -56.0625, 0.354248046875, -2.318359375, 119.25, -1.91796875, -56.09375, 72.125, -18.328125, -49.71875, 62.125, -73.625, -20.859375, -179.25, 169.125, -184.875], [181.0, 96.25, 1.630859375, 155.625, -37.75, -87.0625, 32.90625, 86.375, 55.25, -119.625, 51.0, -38.5625, 210.625, 121.9375, 84.625, 73.875, 108.0], [-79.4375, -34.8125, -59.3125, 65.75, 49.5625, -135.125, -35.28125, -70.4375, -41.21875, 232.125, -164.0, -34.90625, -71.9375, -77.0, 40.71875, 81.125, 3.619140625], [-29.015625, -66.25, 56.09375, -150.625, -223.25, -146.25, -66.125, 36.5, -135.75, -78.6875, -38.6875, 2.744140625, -46.6875, -172.25, 60.78125, -76.5, 36.15625], [-40.25, 36.84375, 11.2265625, -142.0, -55.9375, -38.375, -228.5, 81.4375, 127.6875, 239.75, -35.625, -160.0, 93.375, -99.875, -30.90625, 32.40625, 17.578125], [142.0, -21.046875, -7.51171875, -64.3125, -89.75, 26.0, 108.875, 91.625, 6.36328125, 14.5, 33.40625, -24.765625, -221.0, 85.4375, 152.375, 194.5, -45.0625], [1.07421875, -50.90625, -15.0234375, 20.5, 25.078125, -34.15625, -68.4375, 104.9375, -19.0625, 116.3125, 83.25, -80.8125, 32.90625, 56.28125, 41.3125, -80.5, -29.109375], [-89.625, -81.75, -150.625, -142.0, 147.5, 58.1875, 60.5625, 240.625, 135.625, -76.375, -101.875, 9.7109375, -129.25, -169.375, 3.43359375, -3.1484375, 59.3125], [-115.0625, 61.375, 102.625, -86.375, -24.84375, -38.375, -85.75, 171.5, 14.984375, 87.75, 63.1875, 207.375, -24.8125, 66.125, -186.25, 72.75, 178.625], [-21.015625, -28.53125, -22.640625, -79.75, 30.4375, 83.5, 20.46875, -75.9375, -74.6875, -75.4375, -81.4375, 163.25, -78.875, -105.0625, -71.375, 98.5625, -155.875], [-85.0, -64.3125, 139.875, 30.578125, 44.15625, 16.078125, 87.9375, -52.25, 46.9375, 193.75, -178.125, 22.40625, -97.875, 9.3203125, 22.578125, -91.0, 221.375], [-358.0, 49.78125, -8.6640625, -42.15625, -266.5, -15.3984375, 52.375, 76.5625, -14.21875, -43.40625, 117.5, 145.25, -74.875, 66.4375, -107.375, 143.25, -192.125], [-63.875, -55.71875, 116.75, -54.65625, -78.125, -127.875, -115.6875, 87.8125, 103.375, -84.75, -7.66796875, -38.375, -118.0, -45.59375, 40.03125, 13.171875, 128.25], [-21.0, -231.125, -16.375, -21.09375, -167.75, -1.7041015625, 25.8125, -73.3125, -18.484375, 31.71875, -87.875, 4.39453125, 144.125, -30.71875, 37.5, 38.125, 97.625], [19.828125, 31.578125, 97.375, 155.25, -17.515625, 161.125, -84.6875, 56.15625, 94.5625, -82.875, -154.0, -51.09375, -29.6875, 81.1875, -31.703125, 109.75, -47.40625], [121.0625, 97.875, -184.75, -43.28125, 18.765625, 20.1875, 16.40625, -120.25, -30.59375, 15.8125, 92.6875, 73.375, -26.453125, 59.4375, 15.0703125, 112.75, -51.375], [-25.53125, -64.5, 121.3125, 76.5, 76.375, 174.0, -22.734375, -36.09375, -136.75, 132.25, -22.75, -54.03125, 70.875, 154.375, 109.625, 13.0703125, 55.25], [-28.046875, 85.0, 85.0625, 88.375, -105.375, -9.9921875, -24.15625, -54.9375, 90.0625, 60.3125, 104.1875, -31.5625, -78.1875, 107.875, -186.875, -6.1171875, 36.03125], [143.25, -17.125, 122.375, -172.875, -13.9765625, 60.46875, -72.0625, 197.75, 62.21875, -121.1875, 12.1796875, 174.875, -29.828125, 61.65625, -36.125, -33.71875, -176.625], [-110.1875, -189.25, 44.0, -58.8125, -35.59375, 113.875, 0.595703125, -6.65234375, 29.109375, 59.4375, -176.5, -2.857421875, 91.0625, -32.21875, -84.0625, 62.0, -129.625], [44.0625, -6.046875, 78.0, 108.5, -228.125, -56.125, 62.0625, 67.25, 25.65625, 17.8125, -174.5, -99.375, 64.1875, -78.0, 27.390625, 116.9375, 148.125], [-149.375, 60.46875, 77.5625, -69.625, 87.6875, 27.671875, 26.078125, -98.0625, -5.9296875, 149.375, -6.25, -82.625, -29.421875, -90.3125, -144.75, 28.015625, 28.0], [81.125, 97.6875, -54.40625, -44.0, 98.1875, 17.3125, 17.484375, 67.875, 170.0, 59.34375, -11.8515625, -23.21875, 104.0, -240.75, -10.0546875, 8.3984375, -15.828125], [59.15625, -156.0, 110.625, -82.9375, 49.96875, -31.96875, 31.09375, 155.375, 120.0, -102.5625, 38.0, 88.125, 19.265625, -9.2421875, 158.125, -156.625, 131.125], [51.96875, 65.125, 42.09375, -15.1171875, 59.40625, -111.875, 68.875, -16.109375, -52.125, 216.125, -2.59375, 147.25, -26.484375, 5.5078125, 205.0, -103.5, -183.75], [2.771484375, 42.5625, -312.5, 30.25, 6.4609375, -39.5, -44.0, 38.375, -80.6875, -62.25, 79.5625, -48.59375, -85.5, -18.828125, -172.125, 176.125, -42.34375], [-38.28125, -180.625, 120.1875, -53.03125, -50.125, -62.34375, 53.84375, -45.125, -41.3125, -166.125, 41.4375, 30.03125, -120.75, 128.875, 43.5625, 67.4375, -136.75], [235.5, 3.77734375, 108.875, -70.125, 21.296875, -213.375, -70.375, 123.3125, -211.125, -63.65625, 81.625, -122.5625, 83.25, 6.28125, -95.25, -33.5625, 63.3125], [132.75, 57.65625, -235.0, -63.5625, -119.625, 85.25, 18.859375, -27.6875, 76.375, 128.5, -31.9375, -13.7734375, -109.5625, 80.75, -31.765625, 8.5, 11.09375], [-89.5, 78.3125, -106.875, 6.86328125, 87.5625, -77.375, -126.0625, 59.3125, -140.0, 166.375, 161.75, -46.4375, 72.9375, 6.43359375, 140.0, 36.875, 76.6875], [79.9375, 55.65625, 105.0625, -149.625, -96.125, -72.8125, -31.046875, -52.21875, 99.1875, -38.28125, 131.875, 108.0, 75.0625, -1.6953125, -106.5625, 120.875, -122.0625], [-67.8125, -36.5625, 92.9375, 97.375, -38.5625, 183.875, -143.875, -187.0, -93.6875, 8.1875, -72.5, 42.21875, -223.625, 38.625, -77.75, -42.84375, -134.75], [81.625, 107.9375, 129.25, -31.75, 207.375, 61.25, -36.59375, -184.75, -62.625, -53.9375, -73.25, -157.625, 79.4375, 80.625, 0.853515625, 64.75, -17.828125], [126.4375, 115.9375, -150.0, -75.6875, -133.5, 65.9375, -50.15625, 27.21875, 51.15625, -61.28125, -63.65625, 73.75, 199.875, -45.75, 50.84375, -239.5, 8.140625], [-13.6484375, 120.5, -115.0625, 48.5625, 23.625, -23.140625, -81.9375, -9.59375, 90.125, -92.3125, 66.5, 138.0, -87.8125, 88.375, -107.375, 11.8828125, 126.75], [-9.703125, -3.119140625, 39.46875, 24.328125, -193.875, -35.78125, -202.375, -102.5625, 55.96875, -36.03125, 161.25, 222.5, -38.6875, 77.625, 22.484375, -97.0625, 23.140625], [-35.5625, -3.650390625, -88.4375, -55.40625, -34.21875, -56.96875, -32.84375, -7.3515625, 90.4375, -98.25, -66.8125, -70.0, 168.875, -0.1295166015625, 17.6875, -135.0, 191.5], [44.125, 133.25, 3.587890625, 119.8125, 57.96875, 95.6875, -63.40625, -128.125, -84.5, -58.6875, -151.5, 3.44921875, -5.828125, 66.1875, 184.125, 83.9375, 13.5078125], [15.671875, 34.4375, -309.5, -16.796875, 95.25, -135.25, 40.25, -161.375, 7.5390625, -72.125, 94.625, -11.375, 124.6875, -83.125, 93.0, 151.375, 262.0], [0.85107421875, 41.125, 180.375, -141.0, 111.8125, 204.75, 188.75, -193.375, 36.3125, 45.6875, 115.4375, -59.84375, -27.578125, 85.75, -151.0, 27.546875, -12.8125], [-90.125, -23.453125, -108.9375, -16.78125, -115.1875, 148.5, 33.90625, 17.328125, 69.0625, -104.25, -140.5, 150.625, -203.375, 61.6875, -39.03125, -82.125, -44.59375], [-265.0, 31.3125, -51.90625, 6.76953125, -24.53125, 28.109375, 66.5, 101.8125, -199.375, -49.4375, 49.96875, 3.435546875, 97.9375, -54.59375, 189.5, 171.375, 49.5], [-73.3125, -6.34765625, 5.65625, -38.78125, -112.875, -152.375, 14.6484375, 48.125, 32.0, 160.875, -80.8125, 64.5, -73.0, -56.21875, -45.3125, 128.875, -72.3125], [-102.75, 23.890625, 113.0625, -183.25, -63.0625, 95.125, -172.75, -201.0, -114.25, 16.140625, 139.125, -11.2890625, -24.609375, -45.59375, 60.46875, 66.75, 84.5], [-10.359375, -142.625, 7.65234375, 86.625, 149.5, -16.75, -42.59375, -26.4375, 118.5625, -71.6875, -124.625, 171.5, -24.21875, -70.375, 96.5625, 54.59375, -132.875], [18.109375, 6.1640625, -206.125, 0.6923828125, 124.1875, -78.5625, -78.75, -19.65625, 20.96875, -18.234375, 127.875, 5.5625, -18.875, -104.25, 62.25, -122.25, -222.5], [-116.25, 76.75, 118.125, 270.0, 89.125, 9.9375, -66.0, -77.25, 67.875, -186.25, 138.0, 78.375, 20.671875, 13.875, -34.0625, 10.5078125, -18.890625], [94.125, 47.0625, 2.794921875, 12.609375, 44.28125, -104.4375, -61.0, 98.375, 42.5625, -183.375, 86.6875, -14.1640625, 12.7421875, 165.0, 79.6875, -156.375, -23.125], [66.0, -29.921875, -39.0, -153.125, -69.125, -70.0625, -219.75, -97.9375, 26.78125, -18.3125, 2.9140625, 100.25, 44.25, -50.875, 46.34375, 0.426025390625, 7.58203125], [-88.4375, 37.375, -63.375, 76.25, -31.953125, -46.40625, -139.0, -60.5625, -9.2578125, -80.1875, -193.5, 172.25, 30.546875, 176.75, 121.875, 4.9765625, -107.875], [-47.125, -5.83203125, 134.25, 29.296875, 122.4375, 92.5625, -69.75, 3.75, 28.84375, -105.875, -95.6875, -5.30859375, -139.875, -12.671875, -4.90234375, 27.609375, -125.25], [-167.0, -208.625, 9.796875, 66.625, -86.4375, 44.125, 42.9375, -19.859375, 175.125, 113.5, 27.640625, 26.625, -69.75, 13.1875, -35.28125, -130.875, -7.07421875], [-165.375, -90.75, 136.125, -12.1796875, 98.625, 5.2421875, 86.125, -17.703125, 35.59375, -42.8125, 47.03125, -67.1875, 45.75, 5.27734375, -28.65625, 62.28125, -12.96875]], "dtype": "float16"}, 1], "e": {"__nd__": [238.125, 252.4599609375, 86.515625, 444.41015625, -676.7890625, 13.12890625, -75.82421875, 77.84765625, -442.8203125, -337.390625, 628.0703125, 156.703125, 617.9375, -138.615234375, 313.8125, -133.73828125, -606.107421875, -432.125, -195.72265625, 560.4609375, -70.25634765625, 354.58984375, -272.244140625, -317.6953125, -834.8203125, 153.8125, 21.84375, -200.796875, -295.390625, 306.18096923828125, 462.787109375, -26.2294921875, -169.408203125, 263.546875, -36.828125, 253.1328125, 292.8515625, -1234.0234375, -626.8984375, -799.59375, 53.4765625, 603.0078125, -329.524658203125, 497.365234375, -139.4609375, 791.173828125, 625.546875, 647.78515625, 590.75, 142.40234375, 162.375, 12.734375, -28.40234375, 155.34375, 554.5546875, -512.71484375, 727.453125, -779.72265625, 88.3125, -202.71484375, 31.828125, 437.5625, -474.9521484375, 637.77734375, 149.447265625, 286.9140625, -312.09375, 646.4296875, 190.31591796875, 391.3984375, 109.359375, 191.5078125, 634.6591796875, 798.90625, 702.24609375, 558.376953125, -174.882080078125, 976.099609375, -330.568359375, -1037.802734375, -191.1640625, 381.6328125, 103.64453125, -229.06640625, 464.828125, -474.40625, 266.4296875, -471.5625, -300.93359375, -290.1376953125, 307.84375, 186.7265625, 621.2421875, 131.65625, 161.25, -425.6796875, 117.8125, -191.4921875, 321.3046875, 421.5234375, 349.796875, -530.220703125, -409.09375, -152.80078125, -33.1796875, 306.859375, 105.2265625, -655.75, 194.744140625, -100.234375, 183.4375, -92.431640625, -216.1314697265625, 313.591796875, 129.2890625, 452.78857421875, -487.25, 151.376953125, -256.66796875, -319.4921875, 22.44921875, -503.3935546875, 394.5390625, 143.451171875, -453.562255859375, -200.53125, -173.52734375, -205.33984375, 84.41796875], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(tiled_sum, json.loads(_TESTS), 'triton', *[0.0001, 0.001])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")